In [ ]:
# ============================================================
# STEP 6.28A
# FD002 DATASET + PROTOCOL PRE-FLIGHT
#
# Purpose:
#   Establish a clean FD002 experiment boundary.
#
# IMPORTANT:
#   - TRAIN and TEST sensor files may be loaded.
#   - Official TEST RUL VALUES are NOT read.
#   - No model training yet.
#   - No preprocessing fitted yet.
#   - No threshold selected.
#   - No files written.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28A — FD002 DATASET + PROTOCOL PRE-FLIGHT\n"
    "============================================================"
)


# ============================================================
# 1. Locate project root
# ============================================================

cwd_628a = Path.cwd()


if (cwd_628a / "02_dataset").exists():

    PROJECT_ROOT_628A = cwd_628a

elif (cwd_628a.parent / "02_dataset").exists():

    PROJECT_ROOT_628A = cwd_628a.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


DATA_DIR_628A = (
    PROJECT_ROOT_628A
    /
    "02_dataset"
    /
    "raw"
    /
    "CMAPSSData"
)


print("\n--- PROJECT CONTRACT ---")

print(
    "Project root      :",
    PROJECT_ROOT_628A
)

print(
    "Dataset directory :",
    DATA_DIR_628A
)

print(
    "Dataset exists    :",
    DATA_DIR_628A.exists()
)


if not DATA_DIR_628A.exists():

    raise FileNotFoundError(
        "CMAPSSData directory not found."
    )


# ============================================================
# 2. FD002 source files
# ============================================================

TRAIN_PATH_628A = (
    DATA_DIR_628A
    /
    "train_FD002.txt"
)


TEST_PATH_628A = (
    DATA_DIR_628A
    /
    "test_FD002.txt"
)


RUL_PATH_628A = (
    DATA_DIR_628A
    /
    "RUL_FD002.txt"
)


print("\n--- FD002 FILE CONTRACT ---")

print(
    "TRAIN exists :",
    TRAIN_PATH_628A.exists()
)

print(
    "TEST exists  :",
    TEST_PATH_628A.exists()
)

print(
    "RUL exists   :",
    RUL_PATH_628A.exists()
)


for path_628a in [
    TRAIN_PATH_628A,
    TEST_PATH_628A,
    RUL_PATH_628A,
]:

    if not path_628a.exists():

        raise FileNotFoundError(
            f"Missing required FD002 file: {path_628a}"
        )


# ============================================================
# 3. Column contract
# ============================================================

COLUMN_NAMES_628A = (
    ["unit_id", "cycle"]
    +
    [
        f"op_setting_{i}"
        for i in range(1, 4)
    ]
    +
    [
        f"sensor_{i}"
        for i in range(1, 22)
    ]
)


print("\n--- COLUMN CONTRACT ---")

print(
    "Expected columns :",
    len(
        COLUMN_NAMES_628A
    )
)


if len(
    COLUMN_NAMES_628A
) != 26:

    raise RuntimeError(
        "C-MAPSS column contract must contain 26 columns."
    )


# ============================================================
# 4. Load TRAIN + TEST only
#
# IMPORTANT:
#   RUL_FD002.txt is intentionally NOT read here.
# ============================================================

train_fd002_628a = pd.read_csv(
    TRAIN_PATH_628A,
    sep=r"\s+",
    header=None,
    names=COLUMN_NAMES_628A,
)


test_fd002_628a = pd.read_csv(
    TEST_PATH_628A,
    sep=r"\s+",
    header=None,
    names=COLUMN_NAMES_628A,
)


print("\n--- RAW SHAPES ---")

print(
    "TRAIN shape :",
    train_fd002_628a.shape
)

print(
    "TEST shape  :",
    test_fd002_628a.shape
)


# ============================================================
# 5. Engine counts
# ============================================================

train_engines_628a = int(
    train_fd002_628a[
        "unit_id"
    ].nunique()
)


test_engines_628a = int(
    test_fd002_628a[
        "unit_id"
    ].nunique()
)


print("\n--- ENGINE CONTRACT ---")

print(
    "TRAIN engines :",
    train_engines_628a
)

print(
    "TEST engines  :",
    test_engines_628a
)


# ============================================================
# 6. Duplicate endpoint checks
# ============================================================

train_duplicate_keys_628a = int(
    train_fd002_628a.duplicated(
        [
            "unit_id",
            "cycle",
        ]
    ).sum()
)


test_duplicate_keys_628a = int(
    test_fd002_628a.duplicated(
        [
            "unit_id",
            "cycle",
        ]
    ).sum()
)


print("\n--- KEY INTEGRITY ---")

print(
    "TRAIN duplicate unit/cycle keys :",
    train_duplicate_keys_628a
)

print(
    "TEST duplicate unit/cycle keys  :",
    test_duplicate_keys_628a
)


if train_duplicate_keys_628a != 0:

    raise RuntimeError(
        "Duplicate TRAIN unit/cycle keys detected."
    )


if test_duplicate_keys_628a != 0:

    raise RuntimeError(
        "Duplicate TEST unit/cycle keys detected."
    )


# ============================================================
# 7. Missing / non-finite audit
# ============================================================

train_missing_628a = int(
    train_fd002_628a
    .isna()
    .sum()
    .sum()
)


test_missing_628a = int(
    test_fd002_628a
    .isna()
    .sum()
    .sum()
)


train_numeric_628a = (
    train_fd002_628a
    .to_numpy(
        dtype=float
    )
)


test_numeric_628a = (
    test_fd002_628a
    .to_numpy(
        dtype=float
    )
)


train_all_finite_628a = bool(
    np.isfinite(
        train_numeric_628a
    ).all()
)


test_all_finite_628a = bool(
    np.isfinite(
        test_numeric_628a
    ).all()
)


print("\n--- NUMERICAL INTEGRITY ---")

print(
    "TRAIN missing values :",
    train_missing_628a
)

print(
    "TEST missing values  :",
    test_missing_628a
)

print(
    "TRAIN all finite     :",
    train_all_finite_628a
)

print(
    "TEST all finite      :",
    test_all_finite_628a
)


# ============================================================
# 8. Cycle continuity audit
# ============================================================

def cycle_contract_628a(df):

    rows = []

    for unit_id, group in df.groupby(
        "unit_id",
        sort=True
    ):

        cycles = np.sort(
            group[
                "cycle"
            ].to_numpy(
                dtype=int
            )
        )

        expected = np.arange(
            cycles.min(),
            cycles.max() + 1,
            dtype=int
        )

        rows.append(
            {
                "unit_id":
                    int(unit_id),

                "start_cycle":
                    int(
                        cycles.min()
                    ),

                "end_cycle":
                    int(
                        cycles.max()
                    ),

                "rows":
                    int(
                        len(cycles)
                    ),

                "contiguous":
                    bool(
                        np.array_equal(
                            cycles,
                            expected
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


TRAIN_CYCLE_AUDIT_628A = (
    cycle_contract_628a(
        train_fd002_628a
    )
)


TEST_CYCLE_AUDIT_628A = (
    cycle_contract_628a(
        test_fd002_628a
    )
)


train_noncontiguous_628a = int(
    (
        ~TRAIN_CYCLE_AUDIT_628A[
            "contiguous"
        ]
    ).sum()
)


test_noncontiguous_628a = int(
    (
        ~TEST_CYCLE_AUDIT_628A[
            "contiguous"
        ]
    ).sum()
)


print("\n--- CYCLE CONTINUITY ---")

print(
    "TRAIN non-contiguous engines :",
    train_noncontiguous_628a
)

print(
    "TEST non-contiguous engines  :",
    test_noncontiguous_628a
)


# ============================================================
# 9. Trajectory-length summary
# ============================================================

train_lengths_628a = (
    train_fd002_628a
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
)


test_lengths_628a = (
    test_fd002_628a
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
)


print("\n--- TRAJECTORY LENGTHS ---")

print(
    "TRAIN min / median / max :",
    int(
        train_lengths_628a.min()
    ),
    "/",
    float(
        train_lengths_628a.median()
    ),
    "/",
    int(
        train_lengths_628a.max()
    )
)

print(
    "TEST min / median / max  :",
    int(
        test_lengths_628a.min()
    ),
    "/",
    float(
        test_lengths_628a.median()
    ),
    "/",
    int(
        test_lengths_628a.max()
    )
)


# ============================================================
# 10. Operating-setting audit
#
# We DO NOT infer operating-condition labels yet.
# We only inspect their numerical structure.
# ============================================================

op_cols_628a = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]


OP_SETTING_SUMMARY_628A = (
    train_fd002_628a[
        op_cols_628a
    ]
    .describe()
    .T
)


print("\n--- TRAIN OPERATING-SETTING SUMMARY ---")

print(
    OP_SETTING_SUMMARY_628A[
        [
            "min",
            "mean",
            "std",
            "max",
        ]
    ]
    .to_string()
)


print("\n--- OPERATING-SETTING UNIQUE COUNTS ---")

for col in op_cols_628a:

    print(
        f"{col:<15}:",
        train_fd002_628a[
            col
        ].nunique()
    )


# ============================================================
# 11. Feature variability audit
#
# No features are dropped here.
# This is inspection only.
# ============================================================

candidate_feature_cols_628a = (
    op_cols_628a
    +
    [
        f"sensor_{i}"
        for i in range(1, 22)
    ]
)


FEATURE_VARIABILITY_628A = pd.DataFrame(
    {
        "feature":
            candidate_feature_cols_628a,

        "nunique":
            [
                train_fd002_628a[
                    col
                ].nunique()
                for col in candidate_feature_cols_628a
            ],

        "std":
            [
                float(
                    train_fd002_628a[
                        col
                    ].std()
                )
                for col in candidate_feature_cols_628a
            ],
    }
)


print("\n--- LOWEST-VARIABILITY TRAIN FEATURES ---")

print(
    FEATURE_VARIABILITY_628A
    .sort_values(
        [
            "nunique",
            "std",
        ]
    )
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 12. TEST label seal
# ============================================================

FD002_OFFICIAL_TEST_LABELS_READ_628A = False


print("\n--- TEST-LABEL SEAL ---")

print(
    "RUL_FD002 file exists      :",
    RUL_PATH_628A.exists()
)

print(
    "RUL_FD002 values read      :",
    FD002_OFFICIAL_TEST_LABELS_READ_628A
)

print(
    "Official TEST scored       : False"
)


# ============================================================
# 13. Freeze pre-flight object
# ============================================================

FD002_PREFLIGHT_628A = {

    "dataset":
        "FD002",

    "train_rows":
        int(
            len(
                train_fd002_628a
            )
        ),

    "test_rows":
        int(
            len(
                test_fd002_628a
            )
        ),

    "train_engines":
        train_engines_628a,

    "test_engines":
        test_engines_628a,

    "train_duplicate_keys":
        train_duplicate_keys_628a,

    "test_duplicate_keys":
        test_duplicate_keys_628a,

    "train_noncontiguous_engines":
        train_noncontiguous_628a,

    "test_noncontiguous_engines":
        test_noncontiguous_628a,

    "train_missing":
        train_missing_628a,

    "test_missing":
        test_missing_628a,

    "train_all_finite":
        train_all_finite_628a,

    "test_all_finite":
        test_all_finite_628a,

    "official_test_labels_read":
        False,

    "model_trained":
        False,

    "preprocessing_fitted":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "TRAIN rows / engines        :",
    len(
        train_fd002_628a
    ),
    "/",
    train_engines_628a
)

print(
    "TEST rows / engines         :",
    len(
        test_fd002_628a
    ),
    "/",
    test_engines_628a
)

print(
    "TRAIN duplicate keys        :",
    train_duplicate_keys_628a
)

print(
    "TEST duplicate keys         :",
    test_duplicate_keys_628a
)

print(
    "TRAIN non-contiguous engines:",
    train_noncontiguous_628a
)

print(
    "TEST non-contiguous engines :",
    test_noncontiguous_628a
)

print(
    "TRAIN all finite            :",
    train_all_finite_628a
)

print(
    "TEST all finite             :",
    test_all_finite_628a
)

print(
    "Official TEST labels read   : False"
)

print(
    "Model trained               : False"
)

print(
    "Preprocessing fitted        : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28A ==="
)

In [ ]:
# ============================================================
# STEP 6.28B
# FD002 ENGINE-WISE TRAIN/DEV SPLIT + RUL TARGET CONTRACT
#
# Purpose:
#   Create the immutable FD002 TRAIN/DEV experimental split
#   before fitting any preprocessing or model.
#
# Frozen methodology:
#   - engine-wise 80/20 split
#   - primary target = RUL_cap_125
#   - window length = 30
#
# IMPORTANT:
#   - TEST labels remain unread
#   - no preprocessing fitted
#   - no model trained
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28B — FD002 ENGINE-WISE TRAIN/DEV SPLIT + TARGETS\n"
    "============================================================"
)


# ============================================================
# 1. Required Step-6.28A objects
# ============================================================

required_628b = [
    "train_fd002_628a",
    "test_fd002_628a",
    "FD002_PREFLIGHT_628A",
]


print("\n--- REQUIRED PRE-FLIGHT OBJECTS ---")

for name in required_628b:

    exists = name in globals()

    print(
        f"{name:<28}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required Step-6.28A object: {name}"
        )


# ============================================================
# 2. Frozen methodological constants
# ============================================================

FD002_SPLIT_SEED_628B = 62802
FD002_DEV_FRACTION_628B = 0.20

FD002_WINDOW_LENGTH_628B = 30

FD002_PRIMARY_TARGET_628B = "RUL_cap_125"


print("\n--- FROZEN METHODOLOGY ---")

print(
    "Split unit       : complete engine"
)

print(
    "DEV fraction     :",
    FD002_DEV_FRACTION_628B
)

print(
    "Split seed       :",
    FD002_SPLIT_SEED_628B
)

print(
    "Window length    :",
    FD002_WINDOW_LENGTH_628B
)

print(
    "Primary target   :",
    FD002_PRIMARY_TARGET_628B
)


# ============================================================
# 3. Recover TRAIN engine IDs
# ============================================================

all_train_engine_ids_628b = np.sort(
    train_fd002_628a[
        "unit_id"
    ]
    .unique()
    .astype(int)
)


n_total_engines_628b = int(
    len(
        all_train_engine_ids_628b
    )
)


if n_total_engines_628b != 260:

    raise RuntimeError(
        f"Expected 260 FD002 TRAIN engines, "
        f"found {n_total_engines_628b}."
    )


n_dev_engines_628b = int(
    round(
        n_total_engines_628b
        *
        FD002_DEV_FRACTION_628B
    )
)


n_fit_engines_628b = (
    n_total_engines_628b
    -
    n_dev_engines_628b
)


print("\n--- EXPECTED SPLIT SIZE ---")

print(
    "Total engines :",
    n_total_engines_628b
)

print(
    "FIT engines   :",
    n_fit_engines_628b
)

print(
    "DEV engines   :",
    n_dev_engines_628b
)


# ============================================================
# 4. Deterministic engine-wise split
# ============================================================

rng_628b = np.random.default_rng(
    FD002_SPLIT_SEED_628B
)


permuted_engine_ids_628b = (
    rng_628b.permutation(
        all_train_engine_ids_628b
    )
)


DEV_ENGINE_IDS_628B = np.sort(
    permuted_engine_ids_628b[
        :n_dev_engines_628b
    ]
)


FIT_ENGINE_IDS_628B = np.sort(
    permuted_engine_ids_628b[
        n_dev_engines_628b:
    ]
)


engine_overlap_628b = np.intersect1d(
    FIT_ENGINE_IDS_628B,
    DEV_ENGINE_IDS_628B,
)


print("\n--- ENGINE SPLIT CONTRACT ---")

print(
    "FIT engines         :",
    len(
        FIT_ENGINE_IDS_628B
    )
)

print(
    "DEV engines         :",
    len(
        DEV_ENGINE_IDS_628B
    )
)

print(
    "Engine overlap      :",
    len(
        engine_overlap_628b
    )
)

print(
    "Union = all engines :",
    bool(
        np.array_equal(
            np.sort(
                np.concatenate(
                    [
                        FIT_ENGINE_IDS_628B,
                        DEV_ENGINE_IDS_628B,
                    ]
                )
            ),
            all_train_engine_ids_628b,
        )
    )
)


if len(
    engine_overlap_628b
) != 0:

    raise RuntimeError(
        "FIT/DEV engine leakage detected."
    )


# ============================================================
# 5. Create raw FIT / DEV tables
# ============================================================

fd002_fit_raw_628b = (
    train_fd002_628a.loc[
        train_fd002_628a[
            "unit_id"
        ].isin(
            FIT_ENGINE_IDS_628B
        )
    ]
    .copy()
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .reset_index(
        drop=True
    )
)


fd002_dev_raw_628b = (
    train_fd002_628a.loc[
        train_fd002_628a[
            "unit_id"
        ].isin(
            DEV_ENGINE_IDS_628B
        )
    ]
    .copy()
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n--- RAW SPLIT SHAPES ---")

print(
    "FIT shape :",
    fd002_fit_raw_628b.shape
)

print(
    "DEV shape :",
    fd002_dev_raw_628b.shape
)


print(
    "FIT unique engines :",
    fd002_fit_raw_628b[
        "unit_id"
    ].nunique()
)

print(
    "DEV unique engines :",
    fd002_dev_raw_628b[
        "unit_id"
    ].nunique()
)


# ============================================================
# 6. Training-trajectory RUL construction
#
# C-MAPSS training trajectories run to the simulation
# failure criterion.
#
# Uncapped row RUL:
#
#     max_cycle(engine) - current_cycle
#
# Final training row therefore has RUL_raw = 0.
# ============================================================

def attach_rul_targets_628b(
    df
):

    work = df.copy()


    maximum_cycle = (
        work.groupby(
            "unit_id"
        )[
            "cycle"
        ]
        .transform(
            "max"
        )
    )


    work[
        "RUL_raw"
    ] = (
        maximum_cycle
        -
        work[
            "cycle"
        ]
    ).astype(float)


    work[
        "RUL_cap_125"
    ] = np.minimum(
        work[
            "RUL_raw"
        ],
        125.0,
    )


    work[
        "RUL_cap_130"
    ] = np.minimum(
        work[
            "RUL_raw"
        ],
        130.0,
    )


    return work


FD002_FIT_TARGETED_628B = (
    attach_rul_targets_628b(
        fd002_fit_raw_628b
    )
)


FD002_DEV_TARGETED_628B = (
    attach_rul_targets_628b(
        fd002_dev_raw_628b
    )
)


# ============================================================
# 7. RUL numerical contract
# ============================================================

print("\n--- TARGET CONTRACT ---")


for name_628b, df_628b in [
    (
        "FIT",
        FD002_FIT_TARGETED_628B
    ),
    (
        "DEV",
        FD002_DEV_TARGETED_628B
    ),
]:

    print(
        f"\n{name_628b}"
    )

    for target_628b in [
        "RUL_raw",
        "RUL_cap_125",
        "RUL_cap_130",
    ]:

        values_628b = (
            df_628b[
                target_628b
            ]
            .to_numpy(
                dtype=float
            )
        )


        print(
            f"{target_628b:<12}"
            f" min={values_628b.min():.1f}"
            f" median={np.median(values_628b):.1f}"
            f" max={values_628b.max():.1f}"
            f" finite={np.isfinite(values_628b).all()}"
        )


# ============================================================
# 8. Final-row RUL must equal zero
# ============================================================

def final_row_rul_audit_628b(
    df
):

    final_rows = (
        df.sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .groupby(
            "unit_id",
            as_index=False
        )
        .tail(1)
    )


    return {
        "engines":
            int(
                final_rows[
                    "unit_id"
                ].nunique()
            ),

        "all_raw_zero":
            bool(
                np.allclose(
                    final_rows[
                        "RUL_raw"
                    ].to_numpy(
                        dtype=float
                    ),
                    0.0,
                )
            ),

        "raw_min":
            float(
                final_rows[
                    "RUL_raw"
                ].min()
            ),

        "raw_max":
            float(
                final_rows[
                    "RUL_raw"
                ].max()
            ),
    }


FIT_FINAL_RUL_AUDIT_628B = (
    final_row_rul_audit_628b(
        FD002_FIT_TARGETED_628B
    )
)


DEV_FINAL_RUL_AUDIT_628B = (
    final_row_rul_audit_628b(
        FD002_DEV_TARGETED_628B
    )
)


print("\n--- FINAL-ROW RUL AUDIT ---")

print(
    "FIT:",
    FIT_FINAL_RUL_AUDIT_628B
)

print(
    "DEV:",
    DEV_FINAL_RUL_AUDIT_628B
)


if not FIT_FINAL_RUL_AUDIT_628B[
    "all_raw_zero"
]:

    raise RuntimeError(
        "FIT final-row RUL contract failed."
    )


if not DEV_FINAL_RUL_AUDIT_628B[
    "all_raw_zero"
]:

    raise RuntimeError(
        "DEV final-row RUL contract failed."
    )


# ============================================================
# 9. 30-cycle window eligibility
#
# No windows are created yet.
# We only verify that all selected engines can support
# the frozen 30-cycle sequence length.
# ============================================================

fit_lengths_628b = (
    FD002_FIT_TARGETED_628B
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
)


dev_lengths_628b = (
    FD002_DEV_TARGETED_628B
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
)


fit_window_eligible_628b = int(
    (
        fit_lengths_628b
        >=
        FD002_WINDOW_LENGTH_628B
    ).sum()
)


dev_window_eligible_628b = int(
    (
        dev_lengths_628b
        >=
        FD002_WINDOW_LENGTH_628B
    ).sum()
)


print("\n--- 30-CYCLE WINDOW ELIGIBILITY ---")

print(
    "FIT eligible engines :",
    fit_window_eligible_628b,
    "/",
    len(
        fit_lengths_628b
    )
)

print(
    "DEV eligible engines :",
    dev_window_eligible_628b,
    "/",
    len(
        dev_lengths_628b
    )
)


if fit_window_eligible_628b != len(
    fit_lengths_628b
):

    raise RuntimeError(
        "At least one FIT engine cannot support "
        "a 30-cycle window."
    )


if dev_window_eligible_628b != len(
    dev_lengths_628b
):

    raise RuntimeError(
        "At least one DEV engine cannot support "
        "a 30-cycle window."
    )


# ============================================================
# 10. Exact number of possible sliding windows
#
# For length L and window W:
#
#     windows = L - W + 1
# ============================================================

fit_expected_windows_628b = int(
    (
        fit_lengths_628b
        -
        FD002_WINDOW_LENGTH_628B
        +
        1
    ).sum()
)


dev_expected_windows_628b = int(
    (
        dev_lengths_628b
        -
        FD002_WINDOW_LENGTH_628B
        +
        1
    ).sum()
)


print("\n--- EXPECTED SLIDING WINDOWS ---")

print(
    "FIT expected windows :",
    fit_expected_windows_628b
)

print(
    "DEV expected windows :",
    dev_expected_windows_628b
)


# ============================================================
# 11. Split trajectory-length balance
# ============================================================

print("\n--- TRAJECTORY-LENGTH BALANCE ---")

print(
    "FIT min / median / max :",
    int(
        fit_lengths_628b.min()
    ),
    "/",
    float(
        fit_lengths_628b.median()
    ),
    "/",
    int(
        fit_lengths_628b.max()
    )
)

print(
    "DEV min / median / max :",
    int(
        dev_lengths_628b.min()
    ),
    "/",
    float(
        dev_lengths_628b.median()
    ),
    "/",
    int(
        dev_lengths_628b.max()
    )
)


# ============================================================
# 12. Freeze split metadata
# ============================================================

FD002_SPLIT_CONTRACT_628B = {

    "dataset":
        "FD002",

    "split_unit":
        "engine",

    "split_seed":
        FD002_SPLIT_SEED_628B,

    "fit_engines":
        int(
            len(
                FIT_ENGINE_IDS_628B
            )
        ),

    "dev_engines":
        int(
            len(
                DEV_ENGINE_IDS_628B
            )
        ),

    "engine_overlap":
        0,

    "window_length":
        FD002_WINDOW_LENGTH_628B,

    "primary_target":
        FD002_PRIMARY_TARGET_628B,

    "fit_expected_windows":
        fit_expected_windows_628b,

    "dev_expected_windows":
        dev_expected_windows_628b,

    "official_test_labels_read":
        False,

    "preprocessing_fitted":
        False,

    "model_trained":
        False,
}


# ============================================================
# 13. Final closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FIT engines               :",
    len(
        FIT_ENGINE_IDS_628B
    )
)

print(
    "DEV engines               :",
    len(
        DEV_ENGINE_IDS_628B
    )
)

print(
    "Engine overlap            :",
    len(
        engine_overlap_628b
    )
)

print(
    "Primary target            :",
    FD002_PRIMARY_TARGET_628B
)

print(
    "Window length             :",
    FD002_WINDOW_LENGTH_628B
)

print(
    "FIT expected windows      :",
    fit_expected_windows_628b
)

print(
    "DEV expected windows      :",
    dev_expected_windows_628b
)

print(
    "FIT final-row RUL = 0     :",
    FIT_FINAL_RUL_AUDIT_628B[
        "all_raw_zero"
    ]
)

print(
    "DEV final-row RUL = 0     :",
    DEV_FINAL_RUL_AUDIT_628B[
        "all_raw_zero"
    ]
)

print(
    "Official TEST labels read : False"
)

print(
    "Preprocessing fitted      : False"
)

print(
    "Model trained             : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.28B ==="
)

In [ ]:
# ============================================================
# STEP 6.28C
# FD002 OPERATING-REGIME IDENTIFICATION
# + CONDITION-AWARE NORMALIZATION
#
# Method:
#   1. Fit operational-setting scaler on FIT only
#   2. Fit KMeans(k=6) on FIT only
#   3. Assign regimes to FIT / DEV / TEST
#   4. Fit condition-specific sensor normalization on FIT only
#   5. Fit global op-setting normalization on FIT only
#   6. Transform FIT / DEV / TEST
#
# IMPORTANT:
#   - DEV is transformation-only
#   - TEST is transformation-only
#   - TEST RUL labels remain unread
#   - no RUL model trained
#   - no files written
# ============================================================

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


print(
    "\n"
    "============================================================\n"
    "STEP 6.28C — FD002 CONDITION-AWARE PREPROCESSING\n"
    "============================================================"
)


# ============================================================
# 1. Required Step-6.28B objects
# ============================================================

required_628c = [
    "FD002_FIT_TARGETED_628B",
    "FD002_DEV_TARGETED_628B",
    "test_fd002_628a",
    "FD002_SPLIT_CONTRACT_628B",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_628c:

    exists = name in globals()

    print(
        f"{name:<32}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Frozen feature contract
#
# Same 17-input feature structure retained from FD001.
# ============================================================

FD002_OPERATIONAL_CLUSTER_COLS_628C = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]


FD002_MODEL_OP_COLS_628C = [
    "op_setting_1",
    "op_setting_2",
]


FD002_SENSOR_COLS_628C = [
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21",
]


FD002_N_REGIMES_628C = 6

FD002_REGIME_SEED_628C = 62803


print("\n--- FROZEN FEATURE CONTRACT ---")

print(
    "Operating settings for regime ID :",
    FD002_OPERATIONAL_CLUSTER_COLS_628C
)

print(
    "Model operating inputs           :",
    FD002_MODEL_OP_COLS_628C
)

print(
    "Selected sensor inputs           :",
    len(FD002_SENSOR_COLS_628C)
)

print(
    "Total model features             :",
    (
        len(FD002_MODEL_OP_COLS_628C)
        +
        len(FD002_SENSOR_COLS_628C)
    )
)

print(
    "Operating regimes                :",
    FD002_N_REGIMES_628C
)


if (
    len(FD002_MODEL_OP_COLS_628C)
    +
    len(FD002_SENSOR_COLS_628C)
) != 17:

    raise RuntimeError(
        "Expected exactly 17 model features."
    )


# ============================================================
# 3. Working copies
# ============================================================

fit_raw_628c = (
    FD002_FIT_TARGETED_628B
    .copy()
)


dev_raw_628c = (
    FD002_DEV_TARGETED_628B
    .copy()
)


test_raw_628c = (
    test_fd002_628a
    .copy()
)


# ============================================================
# 4. Fit operational-setting scaler on FIT ONLY
# ============================================================

FD002_REGIME_SCALER_628C = StandardScaler()


fit_op_scaled_628c = (
    FD002_REGIME_SCALER_628C
    .fit_transform(
        fit_raw_628c[
            FD002_OPERATIONAL_CLUSTER_COLS_628C
        ]
    )
)


dev_op_scaled_628c = (
    FD002_REGIME_SCALER_628C
    .transform(
        dev_raw_628c[
            FD002_OPERATIONAL_CLUSTER_COLS_628C
        ]
    )
)


test_op_scaled_628c = (
    FD002_REGIME_SCALER_628C
    .transform(
        test_raw_628c[
            FD002_OPERATIONAL_CLUSTER_COLS_628C
        ]
    )
)


print("\n--- REGIME-SCALER CONTRACT ---")

print(
    "Scaler fitted rows :",
    len(fit_raw_628c)
)

print(
    "DEV used for fit   : False"
)

print(
    "TEST used for fit  : False"
)


# ============================================================
# 5. Fit six-regime KMeans on FIT ONLY
#
# k=6 comes from the FD002 experimental structure,
# not from DEV/TEST tuning.
# ============================================================

FD002_REGIME_KMEANS_628C = KMeans(
    n_clusters=FD002_N_REGIMES_628C,
    random_state=FD002_REGIME_SEED_628C,
    n_init=20,
)


raw_fit_cluster_628c = (
    FD002_REGIME_KMEANS_628C
    .fit_predict(
        fit_op_scaled_628c
    )
)


raw_dev_cluster_628c = (
    FD002_REGIME_KMEANS_628C
    .predict(
        dev_op_scaled_628c
    )
)


raw_test_cluster_628c = (
    FD002_REGIME_KMEANS_628C
    .predict(
        test_op_scaled_628c
    )
)


# ============================================================
# 6. Create deterministic canonical regime IDs
#
# KMeans numeric labels are arbitrary.
#
# We sort raw cluster centers lexicographically in original
# operational-setting units and remap to regime_id 0..5.
# ============================================================

cluster_centers_raw_628c = (
    FD002_REGIME_SCALER_628C
    .inverse_transform(
        FD002_REGIME_KMEANS_628C
        .cluster_centers_
    )
)


center_df_628c = pd.DataFrame(
    cluster_centers_raw_628c,
    columns=FD002_OPERATIONAL_CLUSTER_COLS_628C,
)


center_df_628c[
    "raw_cluster"
] = np.arange(
    FD002_N_REGIMES_628C
)


center_df_628c = (
    center_df_628c
    .sort_values(
        FD002_OPERATIONAL_CLUSTER_COLS_628C
    )
    .reset_index(
        drop=True
    )
)


center_df_628c[
    "regime_id"
] = np.arange(
    FD002_N_REGIMES_628C
)


REGIME_REMAP_628C = dict(
    zip(
        center_df_628c[
            "raw_cluster"
        ].astype(int),

        center_df_628c[
            "regime_id"
        ].astype(int),
    )
)


def canonical_regime_628c(
    raw_labels
):

    return np.asarray(
        [
            REGIME_REMAP_628C[
                int(label)
            ]
            for label in raw_labels
        ],
        dtype=int,
    )


fit_regime_628c = canonical_regime_628c(
    raw_fit_cluster_628c
)


dev_regime_628c = canonical_regime_628c(
    raw_dev_cluster_628c
)


test_regime_628c = canonical_regime_628c(
    raw_test_cluster_628c
)


fit_raw_628c[
    "regime_id"
] = fit_regime_628c


dev_raw_628c[
    "regime_id"
] = dev_regime_628c


test_raw_628c[
    "regime_id"
] = test_regime_628c


print("\n--- CANONICAL OPERATING-REGIME CENTERS ---")

print(
    center_df_628c[
        [
            "regime_id",
            "op_setting_1",
            "op_setting_2",
            "op_setting_3",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Regime population audit
# ============================================================

def regime_counts_628c(
    df,
    population
):

    counts = (
        df[
            "regime_id"
        ]
        .value_counts()
        .sort_index()
        .reindex(
            range(
                FD002_N_REGIMES_628C
            ),
            fill_value=0,
        )
    )


    result = pd.DataFrame(
        {
            "population":
                population,

            "regime_id":
                counts.index.astype(int),

            "rows":
                counts.values.astype(int),
        }
    )


    result[
        "fraction"
    ] = (
        result[
            "rows"
        ]
        /
        result[
            "rows"
        ].sum()
    )


    return result


REGIME_COUNTS_628C = pd.concat(
    [
        regime_counts_628c(
            fit_raw_628c,
            "FIT"
        ),

        regime_counts_628c(
            dev_raw_628c,
            "DEV"
        ),

        regime_counts_628c(
            test_raw_628c,
            "TEST"
        ),
    ],
    ignore_index=True,
)


print("\n--- REGIME POPULATION COUNTS ---")

print(
    REGIME_COUNTS_628C
    .to_string(
        index=False
    )
)


fit_all_regimes_628c = bool(
    (
        REGIME_COUNTS_628C.loc[
            REGIME_COUNTS_628C[
                "population"
            ]
            ==
            "FIT",
            "rows"
        ]
        >
        0
    ).all()
)


if not fit_all_regimes_628c:

    raise RuntimeError(
        "At least one of the six FIT regimes is empty."
    )


# ============================================================
# 8. Distance-to-regime diagnostic
#
# This is descriptive only.
# No threshold or tuning is performed.
# ============================================================

fit_distance_628c = np.min(
    FD002_REGIME_KMEANS_628C.transform(
        fit_op_scaled_628c
    ),
    axis=1,
)


dev_distance_628c = np.min(
    FD002_REGIME_KMEANS_628C.transform(
        dev_op_scaled_628c
    ),
    axis=1,
)


test_distance_628c = np.min(
    FD002_REGIME_KMEANS_628C.transform(
        test_op_scaled_628c
    ),
    axis=1,
)


def distance_summary_628c(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    return {
        "median":
            float(
                np.median(values)
            ),

        "q95":
            float(
                np.quantile(
                    values,
                    0.95
                )
            ),

        "max":
            float(
                np.max(values)
            ),
    }


REGIME_DISTANCE_SUMMARY_628C = {
    "FIT":
        distance_summary_628c(
            fit_distance_628c
        ),

    "DEV":
        distance_summary_628c(
            dev_distance_628c
        ),

    "TEST":
        distance_summary_628c(
            test_distance_628c
        ),
}


print("\n--- REGIME-ASSIGNMENT DISTANCES ---")

for population, values in (
    REGIME_DISTANCE_SUMMARY_628C.items()
):

    print(
        f"{population:<5}"
        f" median={values['median']:.6f}"
        f" q95={values['q95']:.6f}"
        f" max={values['max']:.6f}"
    )


# ============================================================
# 9. Fit condition-specific sensor statistics on FIT ONLY
# ============================================================

sensor_stats_rows_628c = []


for regime_id_628c in range(
    FD002_N_REGIMES_628C
):

    subset_628c = (
        fit_raw_628c.loc[
            fit_raw_628c[
                "regime_id"
            ]
            ==
            regime_id_628c
        ]
    )


    if len(subset_628c) == 0:

        raise RuntimeError(
            f"FIT regime {regime_id_628c} is empty."
        )


    for sensor_628c in (
        FD002_SENSOR_COLS_628C
    ):

        values_628c = (
            subset_628c[
                sensor_628c
            ]
            .to_numpy(
                dtype=float
            )
        )


        sensor_stats_rows_628c.append(
            {
                "regime_id":
                    regime_id_628c,

                "sensor":
                    sensor_628c,

                "n":
                    int(
                        len(values_628c)
                    ),

                "mean":
                    float(
                        np.mean(values_628c)
                    ),

                "std":
                    float(
                        np.std(
                            values_628c,
                            ddof=0
                        )
                    ),
            }
        )


FD002_SENSOR_STATS_628C = pd.DataFrame(
    sensor_stats_rows_628c
)


# ============================================================
# 10. Global FIT fallback sensor statistics
#
# Used only if a regime-specific sensor variance is
# numerically zero / near-zero.
# ============================================================

FD002_GLOBAL_SENSOR_STATS_628C = pd.DataFrame(
    {
        "sensor":
            FD002_SENSOR_COLS_628C,

        "mean":
            [
                float(
                    fit_raw_628c[
                        sensor
                    ].mean()
                )
                for sensor in FD002_SENSOR_COLS_628C
            ],

        "std":
            [
                float(
                    fit_raw_628c[
                        sensor
                    ].std(
                        ddof=0
                    )
                )
                for sensor in FD002_SENSOR_COLS_628C
            ],
    }
)


EPS_628C = 1e-12


GLOBAL_STD_MAP_628C = dict(
    zip(
        FD002_GLOBAL_SENSOR_STATS_628C[
            "sensor"
        ],
        FD002_GLOBAL_SENSOR_STATS_628C[
            "std"
        ],
    )
)


FD002_SENSOR_STATS_628C[
    "fallback_used"
] = False


for idx_628c in (
    FD002_SENSOR_STATS_628C.index
):

    std_628c = float(
        FD002_SENSOR_STATS_628C.loc[
            idx_628c,
            "std"
        ]
    )


    if (
        not np.isfinite(std_628c)
        or
        std_628c <= EPS_628C
    ):

        sensor_name_628c = (
            FD002_SENSOR_STATS_628C.loc[
                idx_628c,
                "sensor"
            ]
        )


        fallback_std_628c = float(
            GLOBAL_STD_MAP_628C[
                sensor_name_628c
            ]
        )


        if (
            not np.isfinite(
                fallback_std_628c
            )
            or
            fallback_std_628c
            <= EPS_628C
        ):

            fallback_std_628c = 1.0


        FD002_SENSOR_STATS_628C.loc[
            idx_628c,
            "std"
        ] = fallback_std_628c


        FD002_SENSOR_STATS_628C.loc[
            idx_628c,
            "fallback_used"
        ] = True


sensor_fallbacks_628c = int(
    FD002_SENSOR_STATS_628C[
        "fallback_used"
    ].sum()
)


print("\n--- SENSOR NORMALIZATION CONTRACT ---")

print(
    "Regime × sensor statistics :",
    FD002_SENSOR_STATS_628C.shape
)

print(
    "Expected rows              :",
    6
    *
    len(FD002_SENSOR_COLS_628C)
)

print(
    "Near-zero-STD fallbacks    :",
    sensor_fallbacks_628c
)


# ============================================================
# 11. Fit model-operating-setting scaler on FIT ONLY
# ============================================================

FD002_MODEL_OP_SCALER_628C = StandardScaler()


FD002_MODEL_OP_SCALER_628C.fit(
    fit_raw_628c[
        FD002_MODEL_OP_COLS_628C
    ]
)


# ============================================================
# 12. Transformation function
#
# Important:
#   Input must already contain regime_id.
#
# Raw source tables remain untouched.
# ============================================================

sensor_stats_lookup_628c = (
    FD002_SENSOR_STATS_628C
    .set_index(
        [
            "regime_id",
            "sensor",
        ]
    )
)


def transform_fd002_628c(
    df
):

    work = df.copy()


    # --------------------------------------------------------
    # Globally standardize the two model operating settings
    # using FIT-only parameters.
    # --------------------------------------------------------

    op_scaled = (
        FD002_MODEL_OP_SCALER_628C
        .transform(
            work[
                FD002_MODEL_OP_COLS_628C
            ]
        )
    )


    for j, op_col in enumerate(
        FD002_MODEL_OP_COLS_628C
    ):

        work[
            f"{op_col}_z"
        ] = (
            op_scaled[
                :,
                j
            ]
        )


    # --------------------------------------------------------
    # Regime-specific sensor normalization
    # --------------------------------------------------------

    for sensor in (
        FD002_SENSOR_COLS_628C
    ):

        normalized = np.empty(
            len(work),
            dtype=float
        )


        for regime_id in range(
            FD002_N_REGIMES_628C
        ):

            mask = (
                work[
                    "regime_id"
                ].to_numpy(
                    dtype=int
                )
                ==
                regime_id
            )


            if not mask.any():
                continue


            mu = float(
                sensor_stats_lookup_628c.loc[
                    (
                        regime_id,
                        sensor,
                    ),
                    "mean"
                ]
            )


            sigma = float(
                sensor_stats_lookup_628c.loc[
                    (
                        regime_id,
                        sensor,
                    ),
                    "std"
                ]
            )


            values = (
                work.loc[
                    mask,
                    sensor
                ]
                .to_numpy(
                    dtype=float
                )
            )


            normalized[
                mask
            ] = (
                values
                -
                mu
            ) / sigma


        work[
            f"{sensor}_z"
        ] = normalized


    return work


# ============================================================
# 13. Transform clean FIT / DEV / TEST
# ============================================================

FD002_FIT_NORMALIZED_628C = (
    transform_fd002_628c(
        fit_raw_628c
    )
)


FD002_DEV_NORMALIZED_628C = (
    transform_fd002_628c(
        dev_raw_628c
    )
)


FD002_TEST_NORMALIZED_628C = (
    transform_fd002_628c(
        test_raw_628c
    )
)


# ============================================================
# 14. Final model-feature contract
# ============================================================

FD002_MODEL_FEATURES_628C = (
    [
        f"{col}_z"
        for col in FD002_MODEL_OP_COLS_628C
    ]
    +
    [
        f"{col}_z"
        for col in FD002_SENSOR_COLS_628C
    ]
)


print("\n--- MODEL FEATURE CONTRACT ---")

print(
    "Feature count :",
    len(
        FD002_MODEL_FEATURES_628C
    )
)

for idx, feature in enumerate(
    FD002_MODEL_FEATURES_628C,
    start=1
):

    print(
        f"{idx:02d}. {feature}"
    )


if len(
    FD002_MODEL_FEATURES_628C
) != 17:

    raise RuntimeError(
        "FD002 normalized feature contract must contain "
        "17 model features."
    )


# ============================================================
# 15. Numerical integrity
# ============================================================

def normalized_integrity_628c(
    df,
    population
):

    values = (
        df[
            FD002_MODEL_FEATURES_628C
        ]
        .to_numpy(
            dtype=float
        )
    )


    return {
        "population":
            population,

        "rows":
            int(
                len(df)
            ),

        "features":
            int(
                values.shape[1]
            ),

        "all_finite":
            bool(
                np.isfinite(
                    values
                ).all()
            ),

        "max_abs":
            float(
                np.max(
                    np.abs(values)
                )
            ),
    }


NORMALIZED_INTEGRITY_628C = pd.DataFrame(
    [
        normalized_integrity_628c(
            FD002_FIT_NORMALIZED_628C,
            "FIT"
        ),

        normalized_integrity_628c(
            FD002_DEV_NORMALIZED_628C,
            "DEV"
        ),

        normalized_integrity_628c(
            FD002_TEST_NORMALIZED_628C,
            "TEST"
        ),
    ]
)


print("\n--- NORMALIZED NUMERICAL INTEGRITY ---")

print(
    NORMALIZED_INTEGRITY_628C
    .to_string(
        index=False
    )
)


if not bool(
    NORMALIZED_INTEGRITY_628C[
        "all_finite"
    ].all()
):

    raise RuntimeError(
        "Non-finite normalized feature detected."
    )


# ============================================================
# 16. FIT normalized-sensor audit by regime
#
# Sensor mean should be approximately 0
# and sensor std approximately 1 within each FIT regime.
# ============================================================

audit_rows_628c = []


for regime_id_628c in range(
    FD002_N_REGIMES_628C
):

    subset_628c = (
        FD002_FIT_NORMALIZED_628C.loc[
            FD002_FIT_NORMALIZED_628C[
                "regime_id"
            ]
            ==
            regime_id_628c
        ]
    )


    for sensor_628c in (
        FD002_SENSOR_COLS_628C
    ):

        normalized_col_628c = (
            f"{sensor_628c}_z"
        )


        audit_rows_628c.append(
            {
                "regime_id":
                    regime_id_628c,

                "sensor":
                    sensor_628c,

                "mean":
                    float(
                        subset_628c[
                            normalized_col_628c
                        ].mean()
                    ),

                "std":
                    float(
                        subset_628c[
                            normalized_col_628c
                        ].std(
                            ddof=0
                        )
                    ),
            }
        )


FIT_NORMALIZATION_AUDIT_628C = (
    pd.DataFrame(
        audit_rows_628c
    )
)


maximum_abs_fit_mean_628c = float(
    FIT_NORMALIZATION_AUDIT_628C[
        "mean"
    ]
    .abs()
    .max()
)


maximum_fit_std_error_628c = float(
    (
        FIT_NORMALIZATION_AUDIT_628C[
            "std"
        ]
        -
        1.0
    )
    .abs()
    .max()
)


print("\n--- FIT NORMALIZATION REPRODUCTION ---")

print(
    "Maximum |regime sensor mean| :",
    maximum_abs_fit_mean_628c
)

print(
    "Maximum |regime sensor std-1|:",
    maximum_fit_std_error_628c
)


# ============================================================
# 17. Freeze preprocessing contract
# ============================================================

FD002_PREPROCESSING_CONTRACT_628C = {

    "dataset":
        "FD002",

    "fit_only_preprocessing":
        True,

    "operating_regimes":
        FD002_N_REGIMES_628C,

    "regime_method":
        "StandardScaler(FIT op settings) + KMeans(k=6)",

    "regime_seed":
        FD002_REGIME_SEED_628C,

    "sensor_normalization":
        "FIT regime-wise z-score",

    "model_operating_normalization":
        "FIT global z-score",

    "model_feature_count":
        len(
            FD002_MODEL_FEATURES_628C
        ),

    "window_length":
        FD002_WINDOW_LENGTH_628B,

    "primary_target":
        FD002_PRIMARY_TARGET_628B,

    "sensor_std_fallbacks":
        sensor_fallbacks_628c,

    "official_test_labels_read":
        False,

    "model_trained":
        False,
}


# ============================================================
# 18. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28C FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Operating regimes             :",
    FD002_N_REGIMES_628C
)

print(
    "Regime model fitted on        : FIT only"
)

print(
    "Condition sensor stats fit on : FIT only"
)

print(
    "Model op scaler fit on        : FIT only"
)

print(
    "Model feature count           :",
    len(
        FD002_MODEL_FEATURES_628C
    )
)

print(
    "Sensor STD fallbacks          :",
    sensor_fallbacks_628c
)

print(
    "FIT normalized finite         :",
    bool(
        NORMALIZED_INTEGRITY_628C.loc[
            NORMALIZED_INTEGRITY_628C[
                "population"
            ]
            ==
            "FIT",
            "all_finite"
        ].iloc[0]
    )
)

print(
    "DEV normalized finite         :",
    bool(
        NORMALIZED_INTEGRITY_628C.loc[
            NORMALIZED_INTEGRITY_628C[
                "population"
            ]
            ==
            "DEV",
            "all_finite"
        ].iloc[0]
    )
)

print(
    "TEST normalized finite        :",
    bool(
        NORMALIZED_INTEGRITY_628C.loc[
            NORMALIZED_INTEGRITY_628C[
                "population"
            ]
            ==
            "TEST",
            "all_finite"
        ].iloc[0]
    )
)

print(
    "Official TEST labels read     : False"
)

print(
    "Model trained                 : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.28C ==="
)

In [ ]:
# ============================================================
# STEP 6.28D0
# FD002 TEST WINDOW-ELIGIBILITY AUDIT
#
# Purpose:
#   Determine exactly how many official FD002 TEST engines
#   contain fewer than the frozen 30 observed cycles.
#
# IMPORTANT:
#   - no TEST RUL labels read
#   - no model trained
#   - no padding rule selected yet
#   - no engines dropped
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28D0 — FD002 TEST WINDOW-ELIGIBILITY AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_628d0 = [
    "FD002_TEST_NORMALIZED_628C",
    "FD002_WINDOW_LENGTH_628B",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_628d0:

    exists = name in globals()

    print(
        f"{name:<32}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Exact TEST trajectory lengths
# ============================================================

TEST_LENGTHS_628D0 = (
    FD002_TEST_NORMALIZED_628C
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
    .sort_index()
)


if len(TEST_LENGTHS_628D0) != 259:

    raise RuntimeError(
        "Expected exactly 259 FD002 TEST engines."
    )


# ============================================================
# 3. Window eligibility
# ============================================================

TEST_SHORT_MASK_628D0 = (
    TEST_LENGTHS_628D0
    <
    FD002_WINDOW_LENGTH_628B
)


TEST_SHORT_ENGINES_628D0 = (
    TEST_LENGTHS_628D0.loc[
        TEST_SHORT_MASK_628D0
    ]
)


n_short_628d0 = int(
    TEST_SHORT_MASK_628D0.sum()
)


n_full_628d0 = int(
    (
        ~TEST_SHORT_MASK_628D0
    ).sum()
)


print("\n--- TEST WINDOW ELIGIBILITY ---")

print(
    "Official TEST engines       :",
    len(TEST_LENGTHS_628D0)
)

print(
    "Window length               :",
    FD002_WINDOW_LENGTH_628B
)

print(
    "Engines with >= 30 cycles   :",
    n_full_628d0
)

print(
    "Engines with < 30 cycles    :",
    n_short_628d0
)

print(
    "Shortest TEST trajectory    :",
    int(
        TEST_LENGTHS_628D0.min()
    )
)

print(
    "Median TEST trajectory      :",
    float(
        TEST_LENGTHS_628D0.median()
    )
)

print(
    "Longest TEST trajectory     :",
    int(
        TEST_LENGTHS_628D0.max()
    )
)


# ============================================================
# 4. Exact short-engine table
# ============================================================

SHORT_TEST_ENGINE_TABLE_628D0 = pd.DataFrame(
    {
        "unit_id":
            TEST_SHORT_ENGINES_628D0.index.astype(int),

        "observed_cycles":
            TEST_SHORT_ENGINES_628D0.values.astype(int),
    }
)


if len(
    SHORT_TEST_ENGINE_TABLE_628D0
) > 0:

    SHORT_TEST_ENGINE_TABLE_628D0[
        "cycles_missing_for_30"
    ] = (
        FD002_WINDOW_LENGTH_628B
        -
        SHORT_TEST_ENGINE_TABLE_628D0[
            "observed_cycles"
        ]
    )


print("\n--- SHORT TEST ENGINES ---")

if len(
    SHORT_TEST_ENGINE_TABLE_628D0
) == 0:

    print(
        "None"
    )

else:

    print(
        SHORT_TEST_ENGINE_TABLE_628D0
        .to_string(
            index=False
        )
    )


# ============================================================
# 5. Length-frequency summary near the boundary
# ============================================================

boundary_counts_628d0 = (
    TEST_LENGTHS_628D0.loc[
        TEST_LENGTHS_628D0
        <=
        40
    ]
    .value_counts()
    .sort_index()
)


TEST_BOUNDARY_LENGTH_COUNTS_628D0 = (
    boundary_counts_628d0
    .rename_axis(
        "trajectory_length"
    )
    .reset_index(
        name="engine_count"
    )
)


print(
    "\n--- TEST LENGTH COUNTS UP TO 40 CYCLES ---"
)

print(
    TEST_BOUNDARY_LENGTH_COUNTS_628D0
    .to_string(
        index=False
    )
)


# ============================================================
# 6. No-loss requirement
# ============================================================

print("\n--- BENCHMARK REQUIREMENT ---")

print(
    "Required final TEST engines : 259"
)

print(
    "Dropping short engines      : NOT ALLOWED"
)

print(
    "Padding rule selected       : False"
)

print(
    "Official TEST labels read   : False"
)


# ============================================================
# 7. Freeze audit object
# ============================================================

FD002_TEST_WINDOW_AUDIT_628D0 = {

    "test_engines":
        int(
            len(
                TEST_LENGTHS_628D0
            )
        ),

    "window_length":
        int(
            FD002_WINDOW_LENGTH_628B
        ),

    "full_length_engines":
        n_full_628d0,

    "short_engines":
        n_short_628d0,

    "minimum_length":
        int(
            TEST_LENGTHS_628D0.min()
        ),

    "padding_rule_selected":
        False,

    "engines_dropped":
        0,

    "official_test_labels_read":
        False,

    "model_trained":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28D0 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "TEST engines             :",
    len(
        TEST_LENGTHS_628D0
    )
)

print(
    "Window length            :",
    FD002_WINDOW_LENGTH_628B
)

print(
    "Engines >= window length :",
    n_full_628d0
)

print(
    "Engines < window length  :",
    n_short_628d0
)

print(
    "Engines dropped          : 0"
)

print(
    "Padding rule selected    : False"
)

print(
    "Official TEST labels read: False"
)

print(
    "Model trained            : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.28D0 ==="
)

In [ ]:
# ============================================================
# STEP 6.28D
# FD002 CLEAN SEQUENCE CONSTRUCTION + CONTRACT FREEZE
#
# Frozen sequence policy:
#
# FIT:
#   all natural 30-cycle sliding windows
#
# DEV:
#   all natural 30-cycle sliding windows
#
# TEST:
#   exactly one endpoint window per engine
#   - if history >= 30 cycles: last 30 observed cycles
#   - if history < 30 cycles:
#       LEFT-PAD by repeating first observed normalized row
#
# Primary benchmark:
#   all 259 TEST engines
#
# Predeclared sensitivity analysis:
#   253 TEST engines requiring no padding
#
# IMPORTANT:
#   - official TEST RUL labels remain unread
#   - no model trained
#   - no threshold selected
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28D — FD002 CLEAN SEQUENCE CONTRACT FREEZE\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_628d = [
    "FD002_FIT_NORMALIZED_628C",
    "FD002_DEV_NORMALIZED_628C",
    "FD002_TEST_NORMALIZED_628C",
    "FD002_MODEL_FEATURES_628C",
    "FD002_WINDOW_LENGTH_628B",
    "FD002_PRIMARY_TARGET_628B",
    "FD002_SPLIT_CONTRACT_628B",
    "FD002_TEST_WINDOW_AUDIT_628D0",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_628d:

    exists = name in globals()

    print(
        f"{name:<34}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Freeze sequence policy
# ============================================================

FD002_TEST_PADDING_POLICY_628D = (
    "left_repeat_first_observed_row"
)

FD002_PRIMARY_TEST_POPULATION_628D = (
    "all_259_engines"
)

FD002_SECONDARY_SENSITIVITY_628D = (
    "253_naturally_eligible_engines"
)


print("\n--- FROZEN SEQUENCE POLICY ---")

print(
    "Window length                :",
    FD002_WINDOW_LENGTH_628B
)

print(
    "Primary target               :",
    FD002_PRIMARY_TARGET_628B
)

print(
    "TEST endpoint rule           : last observed history"
)

print(
    "Short-history padding        :",
    FD002_TEST_PADDING_POLICY_628D
)

print(
    "Primary TEST population      :",
    FD002_PRIMARY_TEST_POPULATION_628D
)

print(
    "Secondary sensitivity        :",
    FD002_SECONDARY_SENSITIVITY_628D
)


# ============================================================
# 3. Natural sliding-window constructor
#
# Used for FIT and DEV.
#
# Target corresponds to the final cycle of each window.
# ============================================================

def build_sliding_windows_628d(
    df,
    feature_cols,
    window_length,
    primary_target,
):

    lengths = (
        df.groupby(
            "unit_id"
        )
        .size()
    )

    expected_windows = int(
        (
            lengths
            -
            window_length
            +
            1
        ).sum()
    )


    n_features = len(
        feature_cols
    )


    X = np.empty(
        (
            expected_windows,
            window_length,
            n_features,
        ),
        dtype=np.float32,
    )


    y_primary = np.empty(
        expected_windows,
        dtype=np.float32,
    )


    y_raw = np.empty(
        expected_windows,
        dtype=np.float32,
    )


    unit_ids = np.empty(
        expected_windows,
        dtype=np.int32,
    )


    endpoint_cycles = np.empty(
        expected_windows,
        dtype=np.int32,
    )


    cursor = 0


    for unit_id, group in df.groupby(
        "unit_id",
        sort=True,
    ):

        group = (
            group
            .sort_values(
                "cycle"
            )
            .reset_index(
                drop=True
            )
        )


        features = (
            group[
                feature_cols
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        cycles = (
            group[
                "cycle"
            ]
            .to_numpy(
                dtype=np.int32
            )
        )


        targets_primary = (
            group[
                primary_target
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        targets_raw = (
            group[
                "RUL_raw"
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        n_windows = (
            len(group)
            -
            window_length
            +
            1
        )


        if n_windows <= 0:
            raise RuntimeError(
                f"Engine {unit_id} cannot support "
                f"a {window_length}-cycle window."
            )


        for start in range(
            n_windows
        ):

            stop = (
                start
                +
                window_length
            )

            endpoint = (
                stop
                -
                1
            )


            X[
                cursor
            ] = features[
                start:stop
            ]


            y_primary[
                cursor
            ] = targets_primary[
                endpoint
            ]


            y_raw[
                cursor
            ] = targets_raw[
                endpoint
            ]


            unit_ids[
                cursor
            ] = int(
                unit_id
            )


            endpoint_cycles[
                cursor
            ] = int(
                cycles[
                    endpoint
                ]
            )


            cursor += 1


    if cursor != expected_windows:

        raise RuntimeError(
            "Sliding-window count mismatch."
        )


    keys = pd.DataFrame(
        {
            "unit_id":
                unit_ids,

            "cycle":
                endpoint_cycles,
        }
    )


    return (
        X,
        y_primary,
        y_raw,
        keys,
    )


# ============================================================
# 4. Construct FIT windows
# ============================================================

(
    X_FIT_FD002_628D,
    Y_FIT_PRIMARY_FD002_628D,
    Y_FIT_RAW_FD002_628D,
    FIT_WINDOW_KEYS_FD002_628D,

) = build_sliding_windows_628d(

    FD002_FIT_NORMALIZED_628C,

    FD002_MODEL_FEATURES_628C,

    FD002_WINDOW_LENGTH_628B,

    FD002_PRIMARY_TARGET_628B,
)


# ============================================================
# 5. Construct DEV windows
# ============================================================

(
    X_DEV_FD002_628D,
    Y_DEV_PRIMARY_FD002_628D,
    Y_DEV_RAW_FD002_628D,
    DEV_WINDOW_KEYS_FD002_628D,

) = build_sliding_windows_628d(

    FD002_DEV_NORMALIZED_628C,

    FD002_MODEL_FEATURES_628C,

    FD002_WINDOW_LENGTH_628B,

    FD002_PRIMARY_TARGET_628B,
)


# ============================================================
# 6. TEST endpoint constructor
#
# Exactly ONE window per official TEST engine.
#
# For short histories:
#
#   [first row repeated] + [entire observed history]
#
# No TEST label is used.
# ============================================================

def build_test_endpoint_windows_628d(
    df,
    feature_cols,
    window_length,
):

    grouped = list(
        df.groupby(
            "unit_id",
            sort=True,
        )
    )


    n_engines = len(
        grouped
    )


    n_features = len(
        feature_cols
    )


    X = np.empty(
        (
            n_engines,
            window_length,
            n_features,
        ),
        dtype=np.float32,
    )


    padding_mask = np.zeros(
        (
            n_engines,
            window_length,
        ),
        dtype=bool,
    )


    metadata_rows = []


    for row_idx, (
        unit_id,
        group,
    ) in enumerate(
        grouped
    ):

        group = (
            group
            .sort_values(
                "cycle"
            )
            .reset_index(
                drop=True
            )
        )


        features = (
            group[
                feature_cols
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        cycles = (
            group[
                "cycle"
            ]
            .to_numpy(
                dtype=np.int32
            )
        )


        observed_length = len(
            group
        )


        if observed_length >= window_length:

            endpoint_window = (
                features[
                    -window_length:
                ]
            )


            padding_count = 0


        else:

            padding_count = (
                window_length
                -
                observed_length
            )


            left_padding = np.repeat(
                features[
                    0:1
                ],
                repeats=padding_count,
                axis=0,
            )


            endpoint_window = np.concatenate(
                [
                    left_padding,
                    features,
                ],
                axis=0,
            )


            padding_mask[
                row_idx,
                :padding_count
            ] = True


        if endpoint_window.shape != (
            window_length,
            n_features,
        ):

            raise RuntimeError(
                f"Endpoint shape failure for "
                f"engine {unit_id}."
            )


        X[
            row_idx
        ] = endpoint_window


        metadata_rows.append(
            {
                "unit_id":
                    int(
                        unit_id
                    ),

                "cycle":
                    int(
                        cycles[-1]
                    ),

                "observed_cycles":
                    int(
                        observed_length
                    ),

                "padding_count":
                    int(
                        padding_count
                    ),

                "padding_used":
                    bool(
                        padding_count > 0
                    ),
            }
        )


    metadata = pd.DataFrame(
        metadata_rows
    )


    return (
        X,
        padding_mask,
        metadata,
    )


# ============================================================
# 7. Construct all 259 TEST endpoint windows
# ============================================================

(
    X_TEST_FD002_628D,
    TEST_PADDING_MASK_FD002_628D,
    TEST_ENDPOINT_META_FD002_628D,

) = build_test_endpoint_windows_628d(

    FD002_TEST_NORMALIZED_628C,

    FD002_MODEL_FEATURES_628C,

    FD002_WINDOW_LENGTH_628B,
)


# ============================================================
# 8. Shape contract
# ============================================================

print("\n--- ARRAY SHAPES ---")

print(
    "FIT X shape  :",
    X_FIT_FD002_628D.shape
)

print(
    "DEV X shape  :",
    X_DEV_FD002_628D.shape
)

print(
    "TEST X shape :",
    X_TEST_FD002_628D.shape
)

print(
    "FIT y shape  :",
    Y_FIT_PRIMARY_FD002_628D.shape
)

print(
    "DEV y shape  :",
    Y_DEV_PRIMARY_FD002_628D.shape
)


expected_fit_628d = (
    FD002_SPLIT_CONTRACT_628B[
        "fit_expected_windows"
    ]
)


expected_dev_628d = (
    FD002_SPLIT_CONTRACT_628B[
        "dev_expected_windows"
    ]
)


if X_FIT_FD002_628D.shape != (
    expected_fit_628d,
    30,
    17,
):

    raise RuntimeError(
        "FIT sequence shape contract failed."
    )


if X_DEV_FD002_628D.shape != (
    expected_dev_628d,
    30,
    17,
):

    raise RuntimeError(
        "DEV sequence shape contract failed."
    )


if X_TEST_FD002_628D.shape != (
    259,
    30,
    17,
):

    raise RuntimeError(
        "TEST sequence shape contract failed."
    )


# ============================================================
# 9. Numerical integrity
# ============================================================

ARRAY_INTEGRITY_628D = pd.DataFrame(
    [
        {
            "object":
                "FIT_X",

            "finite":
                bool(
                    np.isfinite(
                        X_FIT_FD002_628D
                    ).all()
                ),

            "max_abs":
                float(
                    np.abs(
                        X_FIT_FD002_628D
                    ).max()
                ),
        },

        {
            "object":
                "DEV_X",

            "finite":
                bool(
                    np.isfinite(
                        X_DEV_FD002_628D
                    ).all()
                ),

            "max_abs":
                float(
                    np.abs(
                        X_DEV_FD002_628D
                    ).max()
                ),
        },

        {
            "object":
                "TEST_X",

            "finite":
                bool(
                    np.isfinite(
                        X_TEST_FD002_628D
                    ).all()
                ),

            "max_abs":
                float(
                    np.abs(
                        X_TEST_FD002_628D
                    ).max()
                ),
        },

        {
            "object":
                "FIT_y_primary",

            "finite":
                bool(
                    np.isfinite(
                        Y_FIT_PRIMARY_FD002_628D
                    ).all()
                ),

            "max_abs":
                float(
                    np.abs(
                        Y_FIT_PRIMARY_FD002_628D
                    ).max()
                ),
        },

        {
            "object":
                "DEV_y_primary",

            "finite":
                bool(
                    np.isfinite(
                        Y_DEV_PRIMARY_FD002_628D
                    ).all()
                ),

            "max_abs":
                float(
                    np.abs(
                        Y_DEV_PRIMARY_FD002_628D
                    ).max()
                ),
        },
    ]
)


print("\n--- ARRAY NUMERICAL INTEGRITY ---")

print(
    ARRAY_INTEGRITY_628D
    .to_string(
        index=False
    )
)


if not bool(
    ARRAY_INTEGRITY_628D[
        "finite"
    ].all()
):

    raise RuntimeError(
        "Non-finite sequence/target values detected."
    )


# ============================================================
# 10. FIT / DEV target contract
# ============================================================

print("\n--- TARGET CONTRACT ---")

print(
    "Primary target :",
    FD002_PRIMARY_TARGET_628B
)

print(
    "FIT target min / median / max :",
    float(
        Y_FIT_PRIMARY_FD002_628D.min()
    ),
    "/",
    float(
        np.median(
            Y_FIT_PRIMARY_FD002_628D
        )
    ),
    "/",
    float(
        Y_FIT_PRIMARY_FD002_628D.max()
    )
)

print(
    "DEV target min / median / max :",
    float(
        Y_DEV_PRIMARY_FD002_628D.min()
    ),
    "/",
    float(
        np.median(
            Y_DEV_PRIMARY_FD002_628D
        )
    ),
    "/",
    float(
        Y_DEV_PRIMARY_FD002_628D.max()
    )
)


# ============================================================
# 11. Window-key integrity
# ============================================================

fit_duplicate_keys_628d = int(
    FIT_WINDOW_KEYS_FD002_628D
    .duplicated(
        [
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


dev_duplicate_keys_628d = int(
    DEV_WINDOW_KEYS_FD002_628D
    .duplicated(
        [
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


test_duplicate_keys_628d = int(
    TEST_ENDPOINT_META_FD002_628D
    .duplicated(
        [
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


print("\n--- WINDOW-KEY CONTRACT ---")

print(
    "FIT duplicate endpoint keys  :",
    fit_duplicate_keys_628d
)

print(
    "DEV duplicate endpoint keys  :",
    dev_duplicate_keys_628d
)

print(
    "TEST duplicate endpoint keys :",
    test_duplicate_keys_628d
)


if (
    fit_duplicate_keys_628d
    !=
    0
    or
    dev_duplicate_keys_628d
    !=
    0
    or
    test_duplicate_keys_628d
    !=
    0
):

    raise RuntimeError(
        "Duplicate window endpoint keys detected."
    )


# ============================================================
# 12. Padding audit
# ============================================================

padding_engine_count_628d = int(
    TEST_ENDPOINT_META_FD002_628D[
        "padding_used"
    ].sum()
)


padding_total_rows_628d = int(
    TEST_ENDPOINT_META_FD002_628D[
        "padding_count"
    ].sum()
)


print("\n--- TEST PADDING AUDIT ---")

print(
    TEST_ENDPOINT_META_FD002_628D.loc[
        TEST_ENDPOINT_META_FD002_628D[
            "padding_used"
        ],
        [
            "unit_id",
            "observed_cycles",
            "padding_count",
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\nPadded engines :",
    padding_engine_count_628d
)

print(
    "Total padded timesteps :",
    padding_total_rows_628d
)


if (
    padding_engine_count_628d
    !=
    FD002_TEST_WINDOW_AUDIT_628D0[
        "short_engines"
    ]
):

    raise RuntimeError(
        "Padding-engine count does not match D0 audit."
    )


# ============================================================
# 13. Exact padding reproduction check
#
# Verify each artificial left-padding row equals the first
# genuinely observed normalized row for that engine.
# ============================================================

maximum_padding_error_628d = 0.0


for _, meta_row in (
    TEST_ENDPOINT_META_FD002_628D.loc[
        TEST_ENDPOINT_META_FD002_628D[
            "padding_used"
        ]
    ]
    .iterrows()
):

    unit_id = int(
        meta_row[
            "unit_id"
        ]
    )


    padding_count = int(
        meta_row[
            "padding_count"
        ]
    )


    test_position = int(
        TEST_ENDPOINT_META_FD002_628D
        .index[
            TEST_ENDPOINT_META_FD002_628D[
                "unit_id"
            ]
            ==
            unit_id
        ][0]
    )


    first_real = (
        FD002_TEST_NORMALIZED_628C.loc[
            FD002_TEST_NORMALIZED_628C[
                "unit_id"
            ]
            ==
            unit_id,
            FD002_MODEL_FEATURES_628C,
        ]
        .sort_index()
        .iloc[0]
        .to_numpy(
            dtype=np.float32
        )
    )


    padded_rows = (
        X_TEST_FD002_628D[
            test_position,
            :padding_count,
            :
        ]
    )


    error = float(
        np.max(
            np.abs(
                padded_rows
                -
                first_real[
                    None,
                    :
                ]
            )
        )
    )


    maximum_padding_error_628d = max(
        maximum_padding_error_628d,
        error,
    )


print("\n--- PADDING REPRODUCTION ---")

print(
    "Maximum padding reproduction error :",
    maximum_padding_error_628d
)


if maximum_padding_error_628d > 1e-7:

    raise RuntimeError(
        "Padding reproduction check failed."
    )


# ============================================================
# 14. Natural TEST endpoint reproduction
#
# For engines >= 30 cycles, verify endpoint window is
# exactly the last 30 normalized observed rows.
# ============================================================

maximum_natural_window_error_628d = 0.0


for test_position, meta_row in (
    TEST_ENDPOINT_META_FD002_628D
    .iterrows()
):

    if bool(
        meta_row[
            "padding_used"
        ]
    ):
        continue


    unit_id = int(
        meta_row[
            "unit_id"
        ]
    )


    original = (
        FD002_TEST_NORMALIZED_628C.loc[
            FD002_TEST_NORMALIZED_628C[
                "unit_id"
            ]
            ==
            unit_id
        ]
        .sort_values(
            "cycle"
        )[
            FD002_MODEL_FEATURES_628C
        ]
        .to_numpy(
            dtype=np.float32
        )[
            -FD002_WINDOW_LENGTH_628B:
        ]
    )


    error = float(
        np.max(
            np.abs(
                X_TEST_FD002_628D[
                    test_position
                ]
                -
                original
            )
        )
    )


    maximum_natural_window_error_628d = max(
        maximum_natural_window_error_628d,
        error,
    )


print(
    "Maximum natural-window reproduction error :",
    maximum_natural_window_error_628d
)


if maximum_natural_window_error_628d > 1e-7:

    raise RuntimeError(
        "Natural TEST endpoint reproduction failed."
    )


# ============================================================
# 15. Predeclare TEST sensitivity population
# ============================================================

FD002_TEST_NATURAL_MASK_628D = (
    ~TEST_ENDPOINT_META_FD002_628D[
        "padding_used"
    ]
    .to_numpy(
        dtype=bool
    )
)


n_natural_test_628d = int(
    FD002_TEST_NATURAL_MASK_628D.sum()
)


print("\n--- PREDECLARED TEST POPULATIONS ---")

print(
    "Primary TEST population     : 259 engines"
)

print(
    "Natural-window sensitivity  :",
    n_natural_test_628d,
    "engines"
)


if n_natural_test_628d != 253:

    raise RuntimeError(
        "Expected 253 naturally eligible TEST engines."
    )


# ============================================================
# 16. Freeze sequence contract
# ============================================================

FD002_SEQUENCE_CONTRACT_628D = {

    "dataset":
        "FD002",

    "window_length":
        int(
            FD002_WINDOW_LENGTH_628B
        ),

    "feature_count":
        int(
            len(
                FD002_MODEL_FEATURES_628C
            )
        ),

    "primary_target":
        FD002_PRIMARY_TARGET_628B,

    "fit_windows":
        int(
            len(
                X_FIT_FD002_628D
            )
        ),

    "dev_windows":
        int(
            len(
                X_DEV_FD002_628D
            )
        ),

    "test_endpoint_windows":
        int(
            len(
                X_TEST_FD002_628D
            )
        ),

    "test_padding_policy":
        FD002_TEST_PADDING_POLICY_628D,

    "padded_test_engines":
        padding_engine_count_628d,

    "natural_test_engines":
        n_natural_test_628d,

    "primary_test_population":
        "all_259",

    "secondary_test_sensitivity":
        "253_natural_windows",

    "official_test_labels_read":
        False,

    "model_trained":
        False,
}


# ============================================================
# 17. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28D FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FIT windows                 :",
    len(
        X_FIT_FD002_628D
    )
)

print(
    "DEV windows                 :",
    len(
        X_DEV_FD002_628D
    )
)

print(
    "TEST endpoint windows       :",
    len(
        X_TEST_FD002_628D
    )
)

print(
    "Window shape                :",
    X_TEST_FD002_628D.shape[
        1:
    ]
)

print(
    "Primary target              :",
    FD002_PRIMARY_TARGET_628B
)

print(
    "Padded TEST engines         :",
    padding_engine_count_628d
)

print(
    "Natural TEST engines        :",
    n_natural_test_628d
)

print(
    "Padding reproduction exact  :",
    bool(
        maximum_padding_error_628d
        <=
        1e-7
    )
)

print(
    "Natural windows exact       :",
    bool(
        maximum_natural_window_error_628d
        <=
        1e-7
    )
)

print(
    "Official TEST labels read   : False"
)

print(
    "Model trained               : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28D ==="
)

In [ ]:
# ============================================================
# STEP 6.28E0
# RECOVER EXACT FD001 MODEL / TRAINING CONTRACT
#
# Purpose:
#   Read the existing FD001 notebooks AS TEXT ONLY and recover
#   the exact architecture, training hyperparameters, random
#   seeds, loss, optimizer, dropout and MC-dropout settings.
#
# IMPORTANT:
#   - notebooks are NOT executed
#   - no model trained
#   - no TEST labels read
#   - no files written
# ============================================================

from pathlib import Path
import json
import re


print(
    "\n"
    "============================================================\n"
    "STEP 6.28E0 — FD001 MODEL-CONTRACT RECOVERY\n"
    "============================================================"
)


# ============================================================
# 1. Locate project root
# ============================================================

cwd_628e0 = Path.cwd()


if (cwd_628e0 / "03_notebooks").exists():

    PROJECT_ROOT_628E0 = cwd_628e0

elif (cwd_628e0.parent / "03_notebooks").exists():

    PROJECT_ROOT_628E0 = cwd_628e0.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


NOTEBOOK_DIR_628E0 = (
    PROJECT_ROOT_628E0
    /
    "03_notebooks"
)


# ============================================================
# 2. FD001 notebooks most likely to contain the exact
#    predictor and uncertainty contracts
# ============================================================

notebook_paths_628e0 = [

    NOTEBOOK_DIR_628E0
    /
    "03_clean_rul_baselines_fd001.ipynb",

    NOTEBOOK_DIR_628E0
    /
    "06_reliability_signals_fd001.ipynb",
]


print("\n--- SOURCE NOTEBOOKS ---")

for path in notebook_paths_628e0:

    print(
        path.name,
        " | exists =",
        path.exists()
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required FD001 notebook: {path}"
        )


# ============================================================
# 3. Search terms
#
# These cover common PyTorch / TensorFlow formulations
# without assuming which implementation was used.
# ============================================================

search_terms_628e0 = [

    # model architecture
    r"\bclass\b",
    r"nn\.Module",
    r"\bGRU\b",
    r"\bLSTM\b",
    r"Conv1d",
    r"Dense",
    r"Linear",
    r"Dropout",

    # optimization / training
    r"Adam",
    r"optimizer",
    r"learning_rate",
    r"\blr\b",
    r"batch_size",
    r"\bepochs?\b",
    r"patience",
    r"early",
    r"MSELoss",
    r"SmoothL1",
    r"Huber",
    r"criterion",
    r"loss_fn",

    # reproducibility
    r"manual_seed",
    r"random_state",
    r"np\.random\.seed",
    r"random\.seed",

    # uncertainty
    r"mc_dropout",
    r"MC dropout",
    r"mc_std",
    r"n_mc",
    r"mc_samples",
    r"stochastic",
    r"training=True",
]


compiled_terms_628e0 = [
    re.compile(
        term,
        flags=re.IGNORECASE
    )
    for term in search_terms_628e0
]


# ============================================================
# 4. Load notebook code cells
# ============================================================

def load_code_cells_628e0(
    notebook_path
):

    with open(
        notebook_path,
        "r",
        encoding="utf-8",
    ) as f:

        notebook = json.load(f)


    code_cells = []


    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        if cell.get(
            "cell_type"
        ) != "code":

            continue


        source = "".join(
            cell.get(
                "source",
                []
            )
        )


        code_cells.append(
            {
                "cell_index":
                    cell_index,

                "source":
                    source,
            }
        )


    return code_cells


# ============================================================
# 5. Find relevant code cells
# ============================================================

def find_relevant_cells_628e0(
    notebook_path
):

    matches = []


    for cell in load_code_cells_628e0(
        notebook_path
    ):

        source = cell[
            "source"
        ]


        matched_terms = []


        for raw_term, pattern in zip(
            search_terms_628e0,
            compiled_terms_628e0,
        ):

            if pattern.search(
                source
            ):

                matched_terms.append(
                    raw_term
                )


        if matched_terms:

            matches.append(
                {
                    "cell_index":
                        cell[
                            "cell_index"
                        ],

                    "matched_terms":
                        matched_terms,

                    "source":
                        source,
                }
            )


    return matches


# ============================================================
# 6. Score candidate cells
#
# Higher score = more likely to define architecture,
# training or MC-dropout rather than merely reference them.
# ============================================================

def cell_score_628e0(
    source
):

    score = 0


    strong_patterns = {

        r"class\s+\w+.*Module":
            10,

        r"def\s+forward":
            8,

        r"GRU\s*\(":
            8,

        r"LSTM\s*\(":
            8,

        r"Conv1d\s*\(":
            8,

        r"Linear\s*\(":
            5,

        r"Dropout\s*\(":
            5,

        r"optimizer\s*=":
            6,

        r"Adam\s*\(":
            6,

        r"criterion\s*=":
            5,

        r"MSELoss\s*\(":
            5,

        r"SmoothL1":
            5,

        r"batch_size\s*=":
            4,

        r"epochs?\s*=":
            4,

        r"patience\s*=":
            4,

        r"manual_seed":
            3,

        r"mc_std":
            4,

        r"mc_samples":
            5,

        r"n_mc":
            5,

        r"training\s*=\s*True":
            5,
    }


    for pattern, points in (
        strong_patterns.items()
    ):

        if re.search(
            pattern,
            source,
            flags=re.IGNORECASE,
        ):

            score += points


    return score


# ============================================================
# 7. Print compact candidate-cell review
#
# We intentionally print candidate cells rather than the
# entire notebooks.
# ============================================================

FD001_MODEL_CONTRACT_CANDIDATES_628E0 = {}


for notebook_path in (
    notebook_paths_628e0
):

    candidates = (
        find_relevant_cells_628e0(
            notebook_path
        )
    )


    for candidate in candidates:

        candidate[
            "score"
        ] = cell_score_628e0(
            candidate[
                "source"
            ]
        )


    candidates = sorted(
        candidates,
        key=lambda x: (
            -x[
                "score"
            ],
            x[
                "cell_index"
            ],
        )
    )


    FD001_MODEL_CONTRACT_CANDIDATES_628E0[
        notebook_path.name
    ] = candidates


    print(
        "\n"
        "============================================================"
    )

    print(
        "NOTEBOOK:",
        notebook_path.name
    )

    print(
        "============================================================"
    )


    print(
        "Relevant candidate cells:",
        len(
            candidates
        )
    )


    # Show top candidates only to keep VS Code output manageable.
    for candidate in candidates[
        :12
    ]:

        source_lines = (
            candidate[
                "source"
            ]
            .splitlines()
        )


        print(
            "\n------------------------------------------------------------"
        )

        print(
            "CELL INDEX:",
            candidate[
                "cell_index"
            ]
        )

        print(
            "SCORE:",
            candidate[
                "score"
            ]
        )

        print(
            "MATCHED TERMS:",
            candidate[
                "matched_terms"
            ]
        )

        print(
            "------------------------------------------------------------"
        )


        # Maximum 120 lines from a candidate cell.
        for line_number, line in enumerate(
            source_lines[
                :120
            ],
            start=1,
        ):

            print(
                f"{line_number:03d}: {line}"
            )


        if len(
            source_lines
        ) > 120:

            print(
                f"... [trimmed: "
                f"{len(source_lines) - 120} "
                f"additional lines]"
            )


# ============================================================
# 8. Explicit assignment search
#
# Print likely hyperparameter assignment lines from both
# notebooks.
# ============================================================

assignment_regex_628e0 = re.compile(
    r"""
    ^\s*
    (
        [A-Za-z_][A-Za-z0-9_]*
    )
    \s*=\s*
    (
        .+
    )
    $
    """,
    flags=re.VERBOSE,
)


hyperparameter_keywords_628e0 = [

    "seed",
    "batch",
    "epoch",
    "lr",
    "learning",
    "dropout",
    "hidden",
    "gru",
    "lstm",
    "patience",
    "window",
    "mc",
    "sample",
    "target",
]


print(
    "\n"
    "============================================================\n"
    "LIKELY FD001 HYPERPARAMETER ASSIGNMENTS\n"
    "============================================================"
)


FD001_ASSIGNMENT_LINES_628E0 = []


for notebook_path in notebook_paths_628e0:

    for cell in load_code_cells_628e0(
        notebook_path
    ):

        for line in (
            cell[
                "source"
            ]
            .splitlines()
        ):

            match = (
                assignment_regex_628e0
                .match(
                    line
                )
            )


            if match is None:
                continue


            variable = (
                match.group(
                    1
                )
            )


            variable_lower = (
                variable.lower()
            )


            if any(
                keyword
                in
                variable_lower

                for keyword
                in
                hyperparameter_keywords_628e0
            ):

                record = {

                    "notebook":
                        notebook_path.name,

                    "cell":
                        int(
                            cell[
                                "cell_index"
                            ]
                        ),

                    "assignment":
                        line.strip(),
                }


                FD001_ASSIGNMENT_LINES_628E0.append(
                    record
                )


for record in (
    FD001_ASSIGNMENT_LINES_628E0
):

    print(
        f"{record['notebook']} "
        f"| cell {record['cell']} "
        f"| {record['assignment']}"
    )


# ============================================================
# 9. State safeguard
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28E0 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 notebooks executed     : False"
)

print(
    "FD002 model trained          : False"
)

print(
    "Official FD002 TEST labels   : NOT READ"
)

print(
    "Threshold selected           : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28E0 ==="
)

In [ ]:
# ============================================================
# STEP 6.28E-A
# FD002 GRU TRAINING-CONTRACT PRE-FLIGHT
#
# PURPOSE
# -------
# 1. Freeze the FD001-derived GRU training contract for FD002.
# 2. Verify the FD002 window/feature/target contract.
# 3. Recover the exact in-memory FIT / DEV / TEST array names.
#
# IMPORTANT
# ---------
# - NO model training.
# - NO official FD002 TEST labels.
# - NO threshold selection.
# - NO files written.
# ============================================================

import random
import numpy as np
import torch

print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-A — FD002 GRU TRAINING-CONTRACT PRE-FLIGHT\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required frozen FD002 objects
# ------------------------------------------------------------

required_628ea = [
    "FD002_MODEL_FEATURES_628C",
    "FD002_WINDOW_LENGTH_628B",
    "FD002_PRIMARY_TARGET_628B",
    "FD002_SPLIT_CONTRACT_628B",
]

print("\n--- REQUIRED FROZEN FD002 OBJECTS ---")

missing_628ea = []

for name_628ea in required_628ea:

    exists_628ea = (
        name_628ea in globals()
    )

    print(
        f"{name_628ea:<36}: "
        f"{exists_628ea}"
    )

    if not exists_628ea:
        missing_628ea.append(
            name_628ea
        )


if missing_628ea:

    raise RuntimeError(
        "Missing required Step 6.28B/C objects: "
        + str(missing_628ea)
    )


# ------------------------------------------------------------
# 2. Frozen FD001-derived training contract
# ------------------------------------------------------------

FD002_GRU_TRAINING_CONTRACT_628EA = {

    "architecture":
        "single_layer_GRU_regressor",

    "window_length":
        int(FD002_WINDOW_LENGTH_628B),

    "input_size":
        len(FD002_MODEL_FEATURES_628C),

    "hidden_size":
        64,

    "num_layers":
        1,

    "dense_hidden_size":
        32,

    "dropout":
        0.20,

    "target":
        str(FD002_PRIMARY_TARGET_628B),

    "training_batch_size":
        128,

    "validation_batch_size":
        256,

    "optimizer":
        "Adam",

    "learning_rate":
        1e-3,

    "loss":
        "MSELoss",

    "gradient_clip_max_norm":
        1.0,

    "max_epochs":
        100,

    "patience":
        12,

    "seed":
        42,

    "mc_passes":
        50,

    "mc_batch_size":
        512,

    "mc_seed":
        42,
}


print("\n--- FROZEN GRU CONTRACT ---")

for key_628ea, value_628ea in (
    FD002_GRU_TRAINING_CONTRACT_628EA.items()
):

    print(
        f"{key_628ea:<27}: "
        f"{value_628ea}"
    )


# ------------------------------------------------------------
# 3. Critical methodology checks
# ------------------------------------------------------------

window_ok_628ea = (
    int(FD002_WINDOW_LENGTH_628B)
    == 30
)

feature_count_ok_628ea = (
    len(FD002_MODEL_FEATURES_628C)
    == 17
)

target_ok_628ea = (
    str(FD002_PRIMARY_TARGET_628B)
    == "RUL_cap_125"
)


print("\n--- METHODOLOGY CONTRACT CHECKS ---")

print(
    "Window length = 30       :",
    window_ok_628ea
)

print(
    "Model feature count = 17 :",
    feature_count_ok_628ea
)

print(
    "Primary target cap-125   :",
    target_ok_628ea
)


if not all(
    [
        window_ok_628ea,
        feature_count_ok_628ea,
        target_ok_628ea,
    ]
):

    raise RuntimeError(
        "FD002 model contract differs from the "
        "frozen replication methodology."
    )


# ------------------------------------------------------------
# 4. Reproducibility contract
# ------------------------------------------------------------

FD002_TRAINING_SEED_628EA = 42

random.seed(
    FD002_TRAINING_SEED_628EA
)

np.random.seed(
    FD002_TRAINING_SEED_628EA
)

torch.manual_seed(
    FD002_TRAINING_SEED_628EA
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        FD002_TRAINING_SEED_628EA
    )


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE_628EA = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\n--- COMPUTE CONTRACT ---")

print(
    "Device :",
    DEVICE_628EA
)

if DEVICE_628EA.type == "cuda":

    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 5. Recover exact Step 6.28D numerical-array names
# ------------------------------------------------------------

def shape_628ea(obj):

    try:
        return tuple(
            obj.shape
        )

    except Exception:
        return None


expected_shapes_628ea = {

    "FIT_X":
        (37852, 30, 17),

    "DEV_X":
        (8367, 30, 17),

    "TEST_X":
        (259, 30, 17),

    "FIT_Y":
        (37852,),

    "DEV_Y":
        (8367,),
}


matches_628ea = {
    key: []
    for key in expected_shapes_628ea
}


for name_628ea, obj_628ea in list(
    globals().items()
):

    object_shape_628ea = (
        shape_628ea(
            obj_628ea
        )
    )

    for (
        role_628ea,
        expected_shape_628ea
    ) in expected_shapes_628ea.items():

        if (
            object_shape_628ea
            == expected_shape_628ea
        ):

            matches_628ea[
                role_628ea
            ].append(
                name_628ea
            )


print(
    "\n--- STEP 6.28D ARRAY CANDIDATES ---"
)

for (
    role_628ea,
    names_628ea
) in matches_628ea.items():

    print(
        f"\n{role_628ea} "
        f"{expected_shapes_628ea[role_628ea]}"
    )

    if names_628ea:

        for name_628ea in names_628ea:

            print(
                "   ",
                name_628ea
            )

    else:

        print(
            "    NO MATCH FOUND"
        )


# ------------------------------------------------------------
# 6. Relevant named Step 6.28D objects
# ------------------------------------------------------------

print(
    "\n--- ALL FD002 OBJECTS CONTAINING 628D ---"
)

names_628d = sorted(
    name
    for name in globals()
    if (
        "FD002" in name
        and "628D" in name.upper()
    )
)

for name in names_628d:

    obj = globals()[name]

    print(
        f"{name:<42} "
        f"type={type(obj).__name__:<14} "
        f"shape={shape_628ea(obj)}"
    )


# ------------------------------------------------------------
# 7. Final safety statement
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Training contract frozen       : True"
)

print(
    "FD002 model trained            : False"
)

print(
    "Official FD002 TEST labels read: False"
)

print(
    "Threshold selected             : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.28E-A ==="
)

In [ ]:
# ============================================================
# STEP 6.28E-B
# TRAIN FD002 GRU UNDER FROZEN FD001 TRAINING CONTRACT
#
# Frozen methodology:
#   window               = 30
#   features             = 17
#   target               = RUL_cap_125
#   hidden size          = 64
#   GRU layers           = 1
#   dense hidden         = 32
#   dropout              = 0.20
#   batch size           = 128
#   validation batch     = 256
#   optimizer            = Adam
#   learning rate        = 1e-3
#   loss                 = MSE
#   gradient clipping    = 1.0
#   max epochs           = 100
#   patience             = 12
#   seed                 = 42
#
# IMPORTANT:
#   - FIT used for optimization
#   - DEV used for early stopping
#   - official FD002 TEST RUL labels remain unread
#   - TEST is NOT scored
#   - no reliability threshold selected
#   - no files written
# ============================================================

import copy
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-B — TRAIN FD002 GRU\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_628eb = [
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "X_FIT_FD002_628D",
    "X_DEV_FD002_628D",
    "X_TEST_FD002_628D",
    "Y_FIT_PRIMARY_FD002_628D",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_WINDOW_KEYS_FD002_628D",
    "TEST_ENDPOINT_META_FD002_628D",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628eb in required_628eb:

    exists_628eb = (
        name_628eb in globals()
    )

    print(
        f"{name_628eb:<38}: "
        f"{exists_628eb}"
    )

    if not exists_628eb:

        raise RuntimeError(
            f"Missing required object: {name_628eb}"
        )


# ============================================================
# 2. Exact frozen contract
# ============================================================

contract_628eb = (
    FD002_GRU_TRAINING_CONTRACT_628EA
)


SEED_628EB = int(
    contract_628eb["seed"]
)

INPUT_SIZE_628EB = int(
    contract_628eb["input_size"]
)

HIDDEN_SIZE_628EB = int(
    contract_628eb["hidden_size"]
)

NUM_LAYERS_628EB = int(
    contract_628eb["num_layers"]
)

DENSE_SIZE_628EB = int(
    contract_628eb["dense_hidden_size"]
)

DROPOUT_628EB = float(
    contract_628eb["dropout"]
)

TRAIN_BATCH_628EB = int(
    contract_628eb["training_batch_size"]
)

DEV_BATCH_628EB = int(
    contract_628eb["validation_batch_size"]
)

LR_628EB = float(
    contract_628eb["learning_rate"]
)

MAX_EPOCHS_628EB = int(
    contract_628eb["max_epochs"]
)

PATIENCE_628EB = int(
    contract_628eb["patience"]
)

CLIP_NORM_628EB = float(
    contract_628eb["gradient_clip_max_norm"]
)


print("\n--- TRAINING CONTRACT ---")

print(
    "Input size       :",
    INPUT_SIZE_628EB
)

print(
    "Hidden size      :",
    HIDDEN_SIZE_628EB
)

print(
    "GRU layers       :",
    NUM_LAYERS_628EB
)

print(
    "Dense hidden     :",
    DENSE_SIZE_628EB
)

print(
    "Dropout          :",
    DROPOUT_628EB
)

print(
    "Train batch      :",
    TRAIN_BATCH_628EB
)

print(
    "DEV batch        :",
    DEV_BATCH_628EB
)

print(
    "Learning rate    :",
    LR_628EB
)

print(
    "Maximum epochs   :",
    MAX_EPOCHS_628EB
)

print(
    "Patience         :",
    PATIENCE_628EB
)

print(
    "Gradient clip    :",
    CLIP_NORM_628EB
)

print(
    "Seed             :",
    SEED_628EB
)


# ============================================================
# 3. Reproducibility
# ============================================================

random.seed(
    SEED_628EB
)

np.random.seed(
    SEED_628EB
)

torch.manual_seed(
    SEED_628EB
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED_628EB
    )


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE_628EB = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\n--- COMPUTE DEVICE ---")

print(
    "Device :",
    DEVICE_628EB
)


if DEVICE_628EB.type == "cuda":

    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 4. Exact input / target arrays
#
# Explicit PRIMARY targets only.
# ============================================================

X_fit_628eb = np.asarray(
    X_FIT_FD002_628D,
    dtype=np.float32,
)


X_dev_628eb = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32,
)


y_fit_628eb = np.asarray(
    Y_FIT_PRIMARY_FD002_628D,
    dtype=np.float32,
)


y_dev_628eb = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float32,
)


expected_fit_shape_628eb = (
    37852,
    30,
    17,
)


expected_dev_shape_628eb = (
    8367,
    30,
    17,
)


if X_fit_628eb.shape != expected_fit_shape_628eb:

    raise RuntimeError(
        f"Unexpected FIT X shape: "
        f"{X_fit_628eb.shape}"
    )


if X_dev_628eb.shape != expected_dev_shape_628eb:

    raise RuntimeError(
        f"Unexpected DEV X shape: "
        f"{X_dev_628eb.shape}"
    )


if y_fit_628eb.shape != (
    37852,
):

    raise RuntimeError(
        "Unexpected FIT target shape."
    )


if y_dev_628eb.shape != (
    8367,
):

    raise RuntimeError(
        "Unexpected DEV target shape."
    )


print("\n--- ARRAY CONTRACT ---")

print(
    "FIT X :",
    X_fit_628eb.shape
)

print(
    "FIT y :",
    y_fit_628eb.shape
)

print(
    "DEV X :",
    X_dev_628eb.shape
)

print(
    "DEV y :",
    y_dev_628eb.shape
)

print(
    "Target:",
    contract_628eb["target"]
)


# ============================================================
# 5. Torch datasets
# ============================================================

fit_dataset_628eb = TensorDataset(

    torch.from_numpy(
        X_fit_628eb
    ),

    torch.from_numpy(
        y_fit_628eb
    ),
)


dev_dataset_628eb = TensorDataset(

    torch.from_numpy(
        X_dev_628eb
    ),

    torch.from_numpy(
        y_dev_628eb
    ),
)


# Dedicated shuffle generator = reproducible batch order.
shuffle_generator_628eb = (
    torch.Generator()
)

shuffle_generator_628eb.manual_seed(
    SEED_628EB
)


fit_loader_628eb = DataLoader(

    fit_dataset_628eb,

    batch_size=TRAIN_BATCH_628EB,

    shuffle=True,

    num_workers=0,

    pin_memory=(
        DEVICE_628EB.type
        ==
        "cuda"
    ),

    generator=shuffle_generator_628eb,
)


dev_loader_628eb = DataLoader(

    dev_dataset_628eb,

    batch_size=DEV_BATCH_628EB,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE_628EB.type
        ==
        "cuda"
    ),
)


print("\n--- DATA LOADERS ---")

print(
    "FIT batches :",
    len(
        fit_loader_628eb
    )
)

print(
    "DEV batches :",
    len(
        dev_loader_628eb
    )
)


# ============================================================
# 6. Frozen GRU architecture
# ============================================================

class FD002GRURegressor628EB(
    nn.Module
):

    def __init__(
        self,
        input_size=17,
        hidden_size=64,
        num_layers=1,
        dense_size=32,
        dropout=0.20,
    ):

        super().__init__()


        self.gru = nn.GRU(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,
        )


        self.fc1 = nn.Linear(
            hidden_size,
            dense_size,
        )


        self.relu = nn.ReLU()


        self.dropout = nn.Dropout(
            dropout
        )


        self.fc2 = nn.Linear(
            dense_size,
            1,
        )


    def forward(
        self,
        x
    ):

        gru_output, _ = self.gru(
            x
        )


        # Final observed timestep representation
        last_hidden = (
            gru_output[
                :,
                -1,
                :
            ]
        )


        x = self.fc1(
            last_hidden
        )


        x = self.relu(
            x
        )


        x = self.dropout(
            x
        )


        x = self.fc2(
            x
        )


        return x.squeeze(
            -1
        )


# ============================================================
# 7. Instantiate model / optimizer / criterion
# ============================================================

MODEL_FD002_628EB = (
    FD002GRURegressor628EB(

        input_size=INPUT_SIZE_628EB,

        hidden_size=HIDDEN_SIZE_628EB,

        num_layers=NUM_LAYERS_628EB,

        dense_size=DENSE_SIZE_628EB,

        dropout=DROPOUT_628EB,
    )
    .to(
        DEVICE_628EB
    )
)


criterion_628eb = (
    nn.MSELoss()
)


optimizer_628eb = (
    torch.optim.Adam(

        MODEL_FD002_628EB.parameters(),

        lr=LR_628EB,
    )
)


n_parameters_628eb = sum(

    parameter.numel()

    for parameter in (
        MODEL_FD002_628EB.parameters()
    )

    if parameter.requires_grad
)


print("\n--- MODEL CONTRACT ---")

print(
    MODEL_FD002_628EB
)

print(
    "\nTrainable parameters :",
    n_parameters_628eb
)


# ============================================================
# 8. Deterministic DEV evaluation helper
# ============================================================

@torch.no_grad()
def deterministic_predict_628eb(
    model,
    loader,
):

    model.eval()


    predictions = []

    targets = []


    for X_batch, y_batch in loader:

        X_batch = X_batch.to(
            DEVICE_628EB,
            non_blocking=True,
        )


        output = model(
            X_batch
        )


        predictions.append(
            output.detach()
            .cpu()
            .numpy()
        )


        targets.append(
            y_batch.numpy()
        )


    predictions = np.concatenate(
        predictions
    ).astype(
        np.float64
    )


    targets = np.concatenate(
        targets
    ).astype(
        np.float64
    )


    return (
        predictions,
        targets,
    )


def regression_metrics_628eb(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    )


    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    )


    errors = (
        y_pred
        -
        y_true
    )


    mae = float(
        np.mean(
            np.abs(
                errors
            )
        )
    )


    mse = float(
        np.mean(
            errors ** 2
        )
    )


    rmse = float(
        np.sqrt(
            mse
        )
    )


    return {
        "MAE":
            mae,

        "MSE":
            mse,

        "RMSE":
            rmse,

        "mean_error":
            float(
                np.mean(
                    errors
                )
            ),
    }


# ============================================================
# 9. Training loop with frozen early-stopping policy
#
# Best epoch criterion:
#   lowest DEV MSE
# ============================================================

BEST_STATE_FD002_628EB = None

best_dev_mse_628eb = np.inf

best_epoch_628eb = None

epochs_without_improvement_628eb = 0


history_rows_628eb = []


training_start_628eb = time.time()


print(
    "\n--- TRAINING START ---"
)


for epoch_628eb in range(
    1,
    MAX_EPOCHS_628EB + 1,
):

    epoch_start_628eb = (
        time.time()
    )


    MODEL_FD002_628EB.train()


    train_squared_error_sum_628eb = 0.0

    train_absolute_error_sum_628eb = 0.0

    train_count_628eb = 0


    for (
        X_batch_628eb,
        y_batch_628eb
    ) in fit_loader_628eb:

        X_batch_628eb = (
            X_batch_628eb.to(

                DEVICE_628EB,

                non_blocking=True,
            )
        )


        y_batch_628eb = (
            y_batch_628eb.to(

                DEVICE_628EB,

                non_blocking=True,
            )
        )


        optimizer_628eb.zero_grad(
            set_to_none=True
        )


        prediction_628eb = (
            MODEL_FD002_628EB(
                X_batch_628eb
            )
        )


        loss_628eb = (
            criterion_628eb(

                prediction_628eb,

                y_batch_628eb,
            )
        )


        loss_628eb.backward()


        torch.nn.utils.clip_grad_norm_(

            MODEL_FD002_628EB.parameters(),

            max_norm=CLIP_NORM_628EB,
        )


        optimizer_628eb.step()


        batch_error_628eb = (

            prediction_628eb.detach()

            -

            y_batch_628eb
        )


        batch_n_628eb = int(
            y_batch_628eb.shape[0]
        )


        train_squared_error_sum_628eb += float(

            torch.sum(
                batch_error_628eb ** 2
            ).item()
        )


        train_absolute_error_sum_628eb += float(

            torch.sum(
                torch.abs(
                    batch_error_628eb
                )
            ).item()
        )


        train_count_628eb += (
            batch_n_628eb
        )


    train_mse_628eb = (

        train_squared_error_sum_628eb

        /

        train_count_628eb
    )


    train_rmse_628eb = float(
        np.sqrt(
            train_mse_628eb
        )
    )


    train_mae_628eb = (

        train_absolute_error_sum_628eb

        /

        train_count_628eb
    )


    # --------------------------------------------------------
    # DEV inference
    # --------------------------------------------------------

    (
        dev_prediction_epoch_628eb,
        dev_target_epoch_628eb,

    ) = deterministic_predict_628eb(

        MODEL_FD002_628EB,

        dev_loader_628eb,
    )


    dev_metrics_epoch_628eb = (
        regression_metrics_628eb(

            dev_target_epoch_628eb,

            dev_prediction_epoch_628eb,
        )
    )


    dev_mse_628eb = (
        dev_metrics_epoch_628eb[
            "MSE"
        ]
    )


    improved_628eb = (
        dev_mse_628eb
        <
        best_dev_mse_628eb
    )


    if improved_628eb:

        best_dev_mse_628eb = (
            dev_mse_628eb
        )


        best_epoch_628eb = (
            epoch_628eb
        )


        BEST_STATE_FD002_628EB = (
            copy.deepcopy(
                MODEL_FD002_628EB.state_dict()
            )
        )


        epochs_without_improvement_628eb = 0


    else:

        epochs_without_improvement_628eb += 1


    epoch_seconds_628eb = (

        time.time()

        -

        epoch_start_628eb
    )


    history_rows_628eb.append(
        {
            "epoch":
                epoch_628eb,

            "train_MSE":
                train_mse_628eb,

            "train_RMSE":
                train_rmse_628eb,

            "train_MAE":
                train_mae_628eb,

            "dev_MSE":
                dev_metrics_epoch_628eb[
                    "MSE"
                ],

            "dev_RMSE":
                dev_metrics_epoch_628eb[
                    "RMSE"
                ],

            "dev_MAE":
                dev_metrics_epoch_628eb[
                    "MAE"
                ],

            "improved":
                improved_628eb,

            "seconds":
                epoch_seconds_628eb,
        }
    )


    marker_628eb = (
        "  *BEST*"
        if improved_628eb
        else ""
    )


    print(
        f"Epoch "
        f"{epoch_628eb:03d} | "
        f"train MAE="
        f"{train_mae_628eb:8.4f} | "
        f"train RMSE="
        f"{train_rmse_628eb:8.4f} | "
        f"DEV MAE="
        f"{dev_metrics_epoch_628eb['MAE']:8.4f} | "
        f"DEV RMSE="
        f"{dev_metrics_epoch_628eb['RMSE']:8.4f} | "
        f"wait="
        f"{epochs_without_improvement_628eb:02d}/"
        f"{PATIENCE_628EB}"
        f"{marker_628eb}"
    )


    if (
        epochs_without_improvement_628eb
        >=
        PATIENCE_628EB
    ):

        print(
            "\nEarly stopping triggered "
            f"after epoch {epoch_628eb}."
        )

        break


training_seconds_628eb = (

    time.time()

    -

    training_start_628eb
)


# ============================================================
# 10. Restore best DEV-MSE model
# ============================================================

if BEST_STATE_FD002_628EB is None:

    raise RuntimeError(
        "No valid best model state was captured."
    )


MODEL_FD002_628EB.load_state_dict(
    BEST_STATE_FD002_628EB
)


MODEL_FD002_628EB.eval()


TRAINING_HISTORY_FD002_628EB = (
    pd.DataFrame(
        history_rows_628eb
    )
)


print("\n--- TRAINING CLOSURE ---")

print(
    "Epochs executed :",
    len(
        TRAINING_HISTORY_FD002_628EB
    )
)

print(
    "Best epoch      :",
    best_epoch_628eb
)

print(
    "Best DEV MSE    :",
    best_dev_mse_628eb
)

print(
    "Training time s :",
    training_seconds_628eb
)


# ============================================================
# 11. Final deterministic DEV predictions
# ============================================================

(
    DEV_PREDICTIONS_FD002_628EB,
    DEV_TARGETS_FD002_628EB,

) = deterministic_predict_628eb(

    MODEL_FD002_628EB,

    dev_loader_628eb,
)


DEV_ALL_WINDOW_METRICS_FD002_628EB = (
    regression_metrics_628eb(

        DEV_TARGETS_FD002_628EB,

        DEV_PREDICTIONS_FD002_628EB,
    )
)


print(
    "\n--- FINAL DEV ALL-WINDOW METRICS ---"
)

for metric_name_628eb, value_628eb in (
    DEV_ALL_WINDOW_METRICS_FD002_628EB.items()
):

    print(
        f"{metric_name_628eb:<12}: "
        f"{value_628eb:.6f}"
    )


# ============================================================
# 12. Build canonical DEV prediction table
# ============================================================

if len(
    DEV_WINDOW_KEYS_FD002_628D
) != len(
    DEV_PREDICTIONS_FD002_628EB
):

    raise RuntimeError(
        "DEV key/prediction alignment failure."
    )


DEV_PREDICTION_TABLE_FD002_628EB = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
)


DEV_PREDICTION_TABLE_FD002_628EB[
    "target_RUL_cap_125"
] = (
    DEV_TARGETS_FD002_628EB
)


DEV_PREDICTION_TABLE_FD002_628EB[
    "prediction"
] = (
    DEV_PREDICTIONS_FD002_628EB
)


DEV_PREDICTION_TABLE_FD002_628EB[
    "error"
] = (

    DEV_PREDICTION_TABLE_FD002_628EB[
        "prediction"
    ]

    -

    DEV_PREDICTION_TABLE_FD002_628EB[
        "target_RUL_cap_125"
    ]
)


DEV_PREDICTION_TABLE_FD002_628EB[
    "absolute_error"
] = (

    DEV_PREDICTION_TABLE_FD002_628EB[
        "error"
    ]
    .abs()
)


# ============================================================
# 13. DEV endpoint metrics
#
# One final available window per DEV engine.
#
# This gives 52 independent engine endpoints and will later
# support the reliability/uncertainty analysis.
# ============================================================

DEV_ENDPOINT_PREDICTIONS_FD002_628EB = (

    DEV_PREDICTION_TABLE_FD002_628EB

    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )

    .groupby(
        "unit_id",
        as_index=False
    )

    .tail(
        1
    )

    .sort_values(
        "unit_id"
    )

    .reset_index(
        drop=True
    )
)


if len(
    DEV_ENDPOINT_PREDICTIONS_FD002_628EB
) != 52:

    raise RuntimeError(
        "Expected exactly 52 DEV endpoints."
    )


DEV_ENDPOINT_METRICS_FD002_628EB = (
    regression_metrics_628eb(

        DEV_ENDPOINT_PREDICTIONS_FD002_628EB[
            "target_RUL_cap_125"
        ].to_numpy(),

        DEV_ENDPOINT_PREDICTIONS_FD002_628EB[
            "prediction"
        ].to_numpy(),
    )
)


print(
    "\n--- FINAL DEV ENDPOINT METRICS ---"
)

print(
    "DEV endpoint engines :",
    len(
        DEV_ENDPOINT_PREDICTIONS_FD002_628EB
    )
)


for metric_name_628eb, value_628eb in (
    DEV_ENDPOINT_METRICS_FD002_628EB.items()
):

    print(
        f"{metric_name_628eb:<12}: "
        f"{value_628eb:.6f}"
    )


# ============================================================
# 14. Basic prediction sanity
# ============================================================

prediction_min_628eb = float(
    np.min(
        DEV_PREDICTIONS_FD002_628EB
    )
)


prediction_median_628eb = float(
    np.median(
        DEV_PREDICTIONS_FD002_628EB
    )
)


prediction_max_628eb = float(
    np.max(
        DEV_PREDICTIONS_FD002_628EB
    )
)


prediction_finite_628eb = bool(
    np.isfinite(
        DEV_PREDICTIONS_FD002_628EB
    ).all()
)


print(
    "\n--- DEV PREDICTION SANITY ---"
)

print(
    "Finite              :",
    prediction_finite_628eb
)

print(
    "Min prediction      :",
    prediction_min_628eb
)

print(
    "Median prediction   :",
    prediction_median_628eb
)

print(
    "Max prediction      :",
    prediction_max_628eb
)


if not prediction_finite_628eb:

    raise RuntimeError(
        "Non-finite DEV predictions detected."
    )


# ============================================================
# 15. Freeze model evidence object
# ============================================================

FD002_MODEL_EVIDENCE_628EB = {

    "dataset":
        "FD002",

    "model":
        "single_layer_GRU_regressor",

    "training_contract_source":
        "frozen_FD001_contract",

    "target":
        "RUL_cap_125",

    "fit_windows":
        int(
            len(
                X_fit_628eb
            )
        ),

    "dev_windows":
        int(
            len(
                X_dev_628eb
            )
        ),

    "dev_engines":
        int(
            DEV_ENDPOINT_PREDICTIONS_FD002_628EB[
                "unit_id"
            ]
            .nunique()
        ),

    "best_epoch":
        int(
            best_epoch_628eb
        ),

    "epochs_executed":
        int(
            len(
                TRAINING_HISTORY_FD002_628EB
            )
        ),

    "best_dev_mse":
        float(
            best_dev_mse_628eb
        ),

    "dev_window_mae":
        float(
            DEV_ALL_WINDOW_METRICS_FD002_628EB[
                "MAE"
            ]
        ),

    "dev_window_rmse":
        float(
            DEV_ALL_WINDOW_METRICS_FD002_628EB[
                "RMSE"
            ]
        ),

    "dev_endpoint_mae":
        float(
            DEV_ENDPOINT_METRICS_FD002_628EB[
                "MAE"
            ]
        ),

    "dev_endpoint_rmse":
        float(
            DEV_ENDPOINT_METRICS_FD002_628EB[
                "RMSE"
            ]
        ),

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "reliability_threshold_selected":
        False,

    "files_written":
        False,
}


# ============================================================
# 16. Final closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Model                    : single-layer GRU"
)

print(
    "Training source          : FD002 FIT only"
)

print(
    "Early stopping source    : FD002 DEV only"
)

print(
    "Primary target           : RUL_cap_125"
)

print(
    "Best epoch               :",
    best_epoch_628eb
)

print(
    "Epochs executed          :",
    len(
        TRAINING_HISTORY_FD002_628EB
    )
)

print(
    "DEV all-window MAE       :",
    DEV_ALL_WINDOW_METRICS_FD002_628EB[
        "MAE"
    ]
)

print(
    "DEV all-window RMSE      :",
    DEV_ALL_WINDOW_METRICS_FD002_628EB[
        "RMSE"
    ]
)

print(
    "DEV endpoint engines     :",
    len(
        DEV_ENDPOINT_PREDICTIONS_FD002_628EB
    )
)

print(
    "DEV endpoint MAE         :",
    DEV_ENDPOINT_METRICS_FD002_628EB[
        "MAE"
    ]
)

print(
    "DEV endpoint RMSE        :",
    DEV_ENDPOINT_METRICS_FD002_628EB[
        "RMSE"
    ]
)

print(
    "Official TEST labels read: False"
)

print(
    "TEST scored              : False"
)

print(
    "Threshold selected       : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.28E-B ==="
)

In [ ]:
# ============================================================
# STEP 6.28E-C
# FD002 DEV ENDPOINT TARGET-PROVENANCE AUDIT
#
# Purpose:
#   Verify exactly what the 52 DEV endpoint targets represent.
#
# IMPORTANT:
#   - no model retraining
#   - no TEST labels
#   - no threshold selection
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-C — FD002 DEV ENDPOINT PROVENANCE AUDIT\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_628ec = [
    "DEV_ENDPOINT_PREDICTIONS_FD002_628EB",
    "DEV_PREDICTION_TABLE_FD002_628EB",
    "FD002_DEV_NORMALIZED_628C",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_628ec:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing object: {name}"
        )


endpoint_628ec = (
    DEV_ENDPOINT_PREDICTIONS_FD002_628EB
    .copy()
)


# ------------------------------------------------------------
# 2. Structural contract
# ------------------------------------------------------------

print("\n--- ENDPOINT STRUCTURE ---")

print(
    "Endpoint rows       :",
    len(endpoint_628ec)
)

print(
    "Unique DEV engines  :",
    endpoint_628ec[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate engines   :",
    int(
        endpoint_628ec[
            "unit_id"
        ].duplicated().sum()
    )
)


# ------------------------------------------------------------
# 3. Endpoint target distribution
# ------------------------------------------------------------

target_values_628ec = (
    endpoint_628ec[
        "target_RUL_cap_125"
    ]
    .to_numpy(
        dtype=float
    )
)


print("\n--- ENDPOINT TARGET DISTRIBUTION ---")

print(
    "Target minimum      :",
    float(
        target_values_628ec.min()
    )
)

print(
    "Target median       :",
    float(
        np.median(
            target_values_628ec
        )
    )
)

print(
    "Target maximum      :",
    float(
        target_values_628ec.max()
    )
)

print(
    "Unique target count :",
    int(
        np.unique(
            target_values_628ec
        ).size
    )
)

print(
    "Targets equal 0     :",
    int(
        np.sum(
            target_values_628ec
            ==
            0.0
        )
    ),
    "/",
    len(
        target_values_628ec
    )
)


# ------------------------------------------------------------
# 4. Verify these are actual final rows of each DEV engine
# ------------------------------------------------------------

dev_final_cycles_628ec = (
    FD002_DEV_NORMALIZED_628C
    .groupby(
        "unit_id"
    )[
        "cycle"
    ]
    .max()
    .rename(
        "true_final_cycle"
    )
    .reset_index()
)


endpoint_check_628ec = (
    endpoint_628ec[
        [
            "unit_id",
            "cycle",
            "target_RUL_cap_125",
            "prediction",
            "error",
            "absolute_error",
        ]
    ]
    .merge(
        dev_final_cycles_628ec,
        on="unit_id",
        how="left",
        validate="one_to_one",
    )
)


endpoint_check_628ec[
    "is_true_final_cycle"
] = (
    endpoint_check_628ec[
        "cycle"
    ]
    ==
    endpoint_check_628ec[
        "true_final_cycle"
    ]
)


print("\n--- FINAL-CYCLE ALIGNMENT ---")

print(
    "Endpoint = true final cycle :",
    bool(
        endpoint_check_628ec[
            "is_true_final_cycle"
        ].all()
    )
)

print(
    "Aligned engines             :",
    int(
        endpoint_check_628ec[
            "is_true_final_cycle"
        ].sum()
    ),
    "/",
    len(
        endpoint_check_628ec
    )
)


# ------------------------------------------------------------
# 5. Error-sign audit
# ------------------------------------------------------------

errors_628ec = (
    endpoint_check_628ec[
        "error"
    ]
    .to_numpy(
        dtype=float
    )
)


print("\n--- ENDPOINT ERROR SIGN AUDIT ---")

print(
    "Negative errors :",
    int(
        np.sum(
            errors_628ec
            <
            0
        )
    )
)

print(
    "Zero errors     :",
    int(
        np.sum(
            errors_628ec
            ==
            0
        )
    )
)

print(
    "Positive errors :",
    int(
        np.sum(
            errors_628ec
            >
            0
        )
    )
)

print(
    "Mean error      :",
    float(
        np.mean(
            errors_628ec
        )
    )
)

print(
    "MAE             :",
    float(
        np.mean(
            np.abs(
                errors_628ec
            )
        )
    )
)


# ------------------------------------------------------------
# 6. Endpoint prediction distribution
# ------------------------------------------------------------

pred_628ec = (
    endpoint_check_628ec[
        "prediction"
    ]
    .to_numpy(
        dtype=float
    )
)


print("\n--- ENDPOINT PREDICTION DISTRIBUTION ---")

print(
    "Minimum :",
    float(
        pred_628ec.min()
    )
)

print(
    "Median  :",
    float(
        np.median(
            pred_628ec
        )
    )
)

print(
    "Mean    :",
    float(
        pred_628ec.mean()
    )
)

print(
    "Maximum :",
    float(
        pred_628ec.max()
    )
)


# ------------------------------------------------------------
# 7. Freeze interpretation evidence
# ------------------------------------------------------------

FD002_DEV_ENDPOINT_AUDIT_628EC = {

    "endpoint_engines":
        int(
            len(
                endpoint_check_628ec
            )
        ),

    "all_are_true_final_cycles":
        bool(
            endpoint_check_628ec[
                "is_true_final_cycle"
            ].all()
        ),

    "all_targets_zero":
        bool(
            np.all(
                target_values_628ec
                ==
                0.0
            )
        ),

    "positive_error_count":
        int(
            np.sum(
                errors_628ec
                >
                0
            )
        ),

    "endpoint_mae":
        float(
            np.mean(
                np.abs(
                    errors_628ec
                )
            )
        ),

    "endpoint_metric_test_like":
        False,

    "official_test_labels_read":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28E-C FINAL SUMMARY\n"
    "============================================================"
)

print(
    "DEV endpoint engines       :",
    len(
        endpoint_check_628ec
    )
)

print(
    "All true final cycles      :",
    FD002_DEV_ENDPOINT_AUDIT_628EC[
        "all_are_true_final_cycles"
    ]
)

print(
    "All endpoint targets = 0   :",
    FD002_DEV_ENDPOINT_AUDIT_628EC[
        "all_targets_zero"
    ]
)

print(
    "Endpoint MAE               :",
    FD002_DEV_ENDPOINT_AUDIT_628EC[
        "endpoint_mae"
    ]
)

print(
    "Endpoint metric test-like  : False"
)

print(
    "Official TEST labels read  : False"
)

print(
    "Model retrained            : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.28E-C ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-A
# FD002 CLEAN MC-DROPOUT UNCERTAINTY
#
# Frozen FD001 uncertainty contract:
#   MC passes       = 50
#   MC seed         = 42
#   MC batch size   = 512
#   uncertainty     = population STD (ddof=0)
#
# Populations:
#   DEV  : all 8,367 clean windows
#   TEST : 259 endpoint windows
#
# IMPORTANT:
#   - model is NOT retrained
#   - official FD002 TEST RUL labels are NOT read
#   - TEST is NOT scored
#   - no uncertainty normalization yet
#   - no reliability threshold selected
#   - no files written
# ============================================================

import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A — FD002 CLEAN MC-DROPOUT UNCERTAINTY\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_628fa = [
    "MODEL_FD002_628EB",
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "X_DEV_FD002_628D",
    "X_TEST_FD002_628D",
    "DEV_WINDOW_KEYS_FD002_628D",
    "TEST_ENDPOINT_META_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628fa in required_628fa:

    exists_628fa = (
        name_628fa in globals()
    )

    print(
        f"{name_628fa:<40}: "
        f"{exists_628fa}"
    )

    if not exists_628fa:

        raise RuntimeError(
            f"Missing required object: {name_628fa}"
        )


# ============================================================
# 2. Freeze MC-dropout contract
# ============================================================

contract_628fa = (
    FD002_GRU_TRAINING_CONTRACT_628EA
)


MC_PASSES_628FA = int(
    contract_628fa[
        "mc_passes"
    ]
)

MC_BATCH_SIZE_628FA = int(
    contract_628fa[
        "mc_batch_size"
    ]
)

MC_SEED_628FA = int(
    contract_628fa[
        "mc_seed"
    ]
)


print("\n--- FROZEN MC CONTRACT ---")

print(
    "MC passes     :",
    MC_PASSES_628FA
)

print(
    "MC batch size :",
    MC_BATCH_SIZE_628FA
)

print(
    "MC seed       :",
    MC_SEED_628FA
)

print(
    "STD ddof      : 0"
)


if MC_PASSES_628FA != 50:

    raise RuntimeError(
        "Expected frozen MC pass count = 50."
    )


if MC_BATCH_SIZE_628FA != 512:

    raise RuntimeError(
        "Expected frozen MC batch size = 512."
    )


if MC_SEED_628FA != 42:

    raise RuntimeError(
        "Expected frozen MC seed = 42."
    )


# ============================================================
# 3. Device contract
# ============================================================

DEVICE_628FA = next(
    MODEL_FD002_628EB.parameters()
).device


print("\n--- DEVICE ---")

print(
    "Model device :",
    DEVICE_628FA
)


# ============================================================
# 4. Input arrays
# ============================================================

X_dev_628fa = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32,
)


X_test_628fa = np.asarray(
    X_TEST_FD002_628D,
    dtype=np.float32,
)


if X_dev_628fa.shape != (
    8367,
    30,
    17,
):

    raise RuntimeError(
        f"Unexpected DEV shape: "
        f"{X_dev_628fa.shape}"
    )


if X_test_628fa.shape != (
    259,
    30,
    17,
):

    raise RuntimeError(
        f"Unexpected TEST shape: "
        f"{X_test_628fa.shape}"
    )


print("\n--- INPUT CONTRACT ---")

print(
    "DEV X  :",
    X_dev_628fa.shape
)

print(
    "TEST X :",
    X_test_628fa.shape
)


# ============================================================
# 5. Label-free loaders
# ============================================================

dev_mc_dataset_628fa = TensorDataset(
    torch.from_numpy(
        X_dev_628fa
    )
)


test_mc_dataset_628fa = TensorDataset(
    torch.from_numpy(
        X_test_628fa
    )
)


dev_mc_loader_628fa = DataLoader(

    dev_mc_dataset_628fa,

    batch_size=MC_BATCH_SIZE_628FA,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE_628FA.type
        ==
        "cuda"
    ),
)


test_mc_loader_628fa = DataLoader(

    test_mc_dataset_628fa,

    batch_size=MC_BATCH_SIZE_628FA,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        DEVICE_628FA.type
        ==
        "cuda"
    ),
)


# ============================================================
# 6. Deterministic prediction helper
# ============================================================

@torch.inference_mode()
def deterministic_predict_x_628fa(
    model,
    loader,
):

    model.eval()

    predictions_628fa = []


    for (
        X_batch_628fa,
    ) in loader:

        X_batch_628fa = (
            X_batch_628fa.to(
                DEVICE_628FA,
                non_blocking=True,
            )
        )


        pred_628fa = model(
            X_batch_628fa
        )


        predictions_628fa.append(
            pred_628fa
            .detach()
            .cpu()
            .numpy()
        )


    return (
        np.concatenate(
            predictions_628fa
        )
        .astype(
            np.float64
        )
    )


# ============================================================
# 7. Reproduce existing deterministic DEV predictions
#
# This proves that the uncertainty stage begins from the exact
# frozen best model produced in Step 6.28E-B.
# ============================================================

DEV_DETERMINISTIC_RECHECK_628FA = (
    deterministic_predict_x_628fa(

        MODEL_FD002_628EB,

        dev_mc_loader_628fa,
    )
)


dev_reproduction_error_628fa = float(
    np.max(
        np.abs(

            DEV_DETERMINISTIC_RECHECK_628FA

            -

            np.asarray(
                DEV_PREDICTIONS_FD002_628EB,
                dtype=np.float64,
            )
        )
    )
)


print(
    "\n--- DETERMINISTIC DEV REPRODUCTION ---"
)

print(
    "Maximum absolute error :",
    dev_reproduction_error_628fa
)


if dev_reproduction_error_628fa > 1e-6:

    raise RuntimeError(
        "Frozen DEV predictions were not reproduced."
    )


# ============================================================
# 8. Deterministic label-free TEST predictions
# ============================================================

TEST_DETERMINISTIC_PREDICTIONS_628FA = (
    deterministic_predict_x_628fa(

        MODEL_FD002_628EB,

        test_mc_loader_628fa,
    )
)


if not np.isfinite(
    TEST_DETERMINISTIC_PREDICTIONS_628FA
).all():

    raise RuntimeError(
        "Non-finite TEST predictions detected."
    )


print("\n--- LABEL-FREE TEST PREDICTIONS ---")

print(
    "Count  :",
    len(
        TEST_DETERMINISTIC_PREDICTIONS_628FA
    )
)

print(
    "Minimum:",
    float(
        np.min(
            TEST_DETERMINISTIC_PREDICTIONS_628FA
        )
    )
)

print(
    "Median :",
    float(
        np.median(
            TEST_DETERMINISTIC_PREDICTIONS_628FA
        )
    )
)

print(
    "Maximum:",
    float(
        np.max(
            TEST_DETERMINISTIC_PREDICTIONS_628FA
        )
    )
)


# ============================================================
# 9. MC-dropout inference helper
#
# MODEL.train() activates the frozen Dropout layer.
#
# This model contains:
#   - no BatchNorm
#   - one GRU layer
#
# Thus train mode activates the explicit downstream dropout
# used for the FD001 uncertainty mechanism.
# ============================================================

@torch.inference_mode()
def mc_dropout_predict_628fa(
    model,
    loader,
    n_samples,
    seed,
):

    # --------------------------------------------------------
    # Exact reproducible stochastic stream
    # --------------------------------------------------------

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    n_observations_628fa = len(
        loader.dataset
    )


    mc_matrix_628fa = np.empty(
        (
            n_samples,
            n_observations_628fa,
        ),
        dtype=np.float32,
    )


    # Activates dropout.
    model.train()


    for sample_idx_628fa in range(
        n_samples
    ):

        pass_predictions_628fa = []


        for (
            X_batch_628fa,
        ) in loader:

            X_batch_628fa = (
                X_batch_628fa.to(
                    DEVICE_628FA,
                    non_blocking=True,
                )
            )


            prediction_628fa = model(
                X_batch_628fa
            )


            pass_predictions_628fa.append(
                prediction_628fa
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


        pass_predictions_628fa = (
            np.concatenate(
                pass_predictions_628fa
            )
        )


        if len(
            pass_predictions_628fa
        ) != n_observations_628fa:

            raise RuntimeError(
                "MC prediction-count mismatch."
            )


        mc_matrix_628fa[
            sample_idx_628fa
        ] = (
            pass_predictions_628fa
        )


    # Restore inference mode.
    model.eval()


    mc_mean_628fa = (
        mc_matrix_628fa
        .mean(
            axis=0
        )
        .astype(
            np.float64
        )
    )


    mc_std_628fa = (
        mc_matrix_628fa
        .std(
            axis=0,
            ddof=0,
        )
        .astype(
            np.float64
        )
    )


    return (
        mc_matrix_628fa,
        mc_mean_628fa,
        mc_std_628fa,
    )


# ============================================================
# 10. MC-dropout on all clean DEV windows
# ============================================================

print(
    "\n--- MC-DROPOUT: CLEAN DEV ---"
)

print(
    "Running",
    MC_PASSES_628FA,
    "stochastic passes..."
)


(
    DEV_MC_MATRIX_FD002_628FA,
    DEV_MC_MEAN_FD002_628FA,
    DEV_MC_STD_FD002_628FA,

) = mc_dropout_predict_628fa(

    MODEL_FD002_628EB,

    dev_mc_loader_628fa,

    n_samples=MC_PASSES_628FA,

    seed=MC_SEED_628FA,
)


# ============================================================
# 11. MC-dropout on official TEST endpoints
#
# Still LABEL-FREE.
# ============================================================

print(
    "\n--- MC-DROPOUT: LABEL-FREE TEST ---"
)

print(
    "Running",
    MC_PASSES_628FA,
    "stochastic passes..."
)


(
    TEST_MC_MATRIX_FD002_628FA,
    TEST_MC_MEAN_FD002_628FA,
    TEST_MC_STD_FD002_628FA,

) = mc_dropout_predict_628fa(

    MODEL_FD002_628EB,

    test_mc_loader_628fa,

    n_samples=MC_PASSES_628FA,

    seed=MC_SEED_628FA,
)


# ============================================================
# 12. Numerical integrity
# ============================================================

uncertainty_integrity_628fa = pd.DataFrame(
    [
        {
            "population":
                "DEV",

            "n":
                len(
                    DEV_MC_STD_FD002_628FA
                ),

            "finite_mean":
                bool(
                    np.isfinite(
                        DEV_MC_MEAN_FD002_628FA
                    ).all()
                ),

            "finite_std":
                bool(
                    np.isfinite(
                        DEV_MC_STD_FD002_628FA
                    ).all()
                ),

            "negative_std_count":
                int(
                    np.sum(
                        DEV_MC_STD_FD002_628FA
                        <
                        0
                    )
                ),
        },

        {
            "population":
                "TEST",

            "n":
                len(
                    TEST_MC_STD_FD002_628FA
                ),

            "finite_mean":
                bool(
                    np.isfinite(
                        TEST_MC_MEAN_FD002_628FA
                    ).all()
                ),

            "finite_std":
                bool(
                    np.isfinite(
                        TEST_MC_STD_FD002_628FA
                    ).all()
                ),

            "negative_std_count":
                int(
                    np.sum(
                        TEST_MC_STD_FD002_628FA
                        <
                        0
                    )
                ),
        },
    ]
)


print("\n--- MC NUMERICAL INTEGRITY ---")

print(
    uncertainty_integrity_628fa
    .to_string(
        index=False
    )
)


if not (
    uncertainty_integrity_628fa[
        "finite_mean"
    ].all()
    and
    uncertainty_integrity_628fa[
        "finite_std"
    ].all()
):

    raise RuntimeError(
        "Non-finite MC-dropout output detected."
    )


# ============================================================
# 13. Raw uncertainty distributions
# ============================================================

def uncertainty_summary_628fa(
    values
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )


    return {
        "minimum":
            float(
                np.min(
                    values
                )
            ),

        "q25":
            float(
                np.quantile(
                    values,
                    0.25,
                )
            ),

        "median":
            float(
                np.median(
                    values
                )
            ),

        "q75":
            float(
                np.quantile(
                    values,
                    0.75,
                )
            ),

        "maximum":
            float(
                np.max(
                    values
                )
            ),

        "mean":
            float(
                np.mean(
                    values
                )
            ),
    }


DEV_MC_SUMMARY_FD002_628FA = (
    uncertainty_summary_628fa(
        DEV_MC_STD_FD002_628FA
    )
)


TEST_MC_SUMMARY_FD002_628FA = (
    uncertainty_summary_628fa(
        TEST_MC_STD_FD002_628FA
    )
)


print("\n--- RAW MC-STD DISTRIBUTIONS ---")

print("\nDEV:")

for key_628fa, value_628fa in (
    DEV_MC_SUMMARY_FD002_628FA.items()
):

    print(
        f"{key_628fa:<8}: "
        f"{value_628fa:.9f}"
    )


print("\nTEST:")

for key_628fa, value_628fa in (
    TEST_MC_SUMMARY_FD002_628FA.items()
):

    print(
        f"{key_628fa:<8}: "
        f"{value_628fa:.9f}"
    )


# ============================================================
# 14. MC mean vs deterministic prediction
#
# These do NOT have to be identical.
# This is descriptive only.
# ============================================================

dev_mcmean_det_mae_628fa = float(
    np.mean(
        np.abs(

            DEV_MC_MEAN_FD002_628FA

            -

            DEV_DETERMINISTIC_RECHECK_628FA
        )
    )
)


test_mcmean_det_mae_628fa = float(
    np.mean(
        np.abs(

            TEST_MC_MEAN_FD002_628FA

            -

            TEST_DETERMINISTIC_PREDICTIONS_628FA
        )
    )
)


print(
    "\n--- MC MEAN VS DETERMINISTIC ---"
)

print(
    "DEV mean absolute difference  :",
    dev_mcmean_det_mae_628fa
)

print(
    "TEST mean absolute difference :",
    test_mcmean_det_mae_628fa
)


# ============================================================
# 15. Canonical clean DEV uncertainty table
# ============================================================

if len(
    DEV_WINDOW_KEYS_FD002_628D
) != len(
    DEV_MC_STD_FD002_628FA
):

    raise RuntimeError(
        "DEV uncertainty/key alignment failure."
    )


DEV_CLEAN_UNCERTAINTY_FD002_628FA = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
)


DEV_CLEAN_UNCERTAINTY_FD002_628FA[
    "deterministic_prediction"
] = (
    DEV_DETERMINISTIC_RECHECK_628FA
)


DEV_CLEAN_UNCERTAINTY_FD002_628FA[
    "mc_mean"
] = (
    DEV_MC_MEAN_FD002_628FA
)


DEV_CLEAN_UNCERTAINTY_FD002_628FA[
    "mc_std"
] = (
    DEV_MC_STD_FD002_628FA
)


# ============================================================
# 16. Canonical label-free TEST uncertainty table
# ============================================================

TEST_CLEAN_UNCERTAINTY_FD002_628FA = (
    TEST_ENDPOINT_META_FD002_628D
    .copy()
)


TEST_CLEAN_UNCERTAINTY_FD002_628FA[
    "deterministic_prediction"
] = (
    TEST_DETERMINISTIC_PREDICTIONS_628FA
)


TEST_CLEAN_UNCERTAINTY_FD002_628FA[
    "mc_mean"
] = (
    TEST_MC_MEAN_FD002_628FA
)


TEST_CLEAN_UNCERTAINTY_FD002_628FA[
    "mc_std"
] = (
    TEST_MC_STD_FD002_628FA
)


# ============================================================
# 17. Freeze uncertainty contract
# ============================================================

FD002_MC_CONTRACT_628FA = {

    "dataset":
        "FD002",

    "mc_passes":
        MC_PASSES_628FA,

    "mc_batch_size":
        MC_BATCH_SIZE_628FA,

    "mc_seed":
        MC_SEED_628FA,

    "std_ddof":
        0,

    "dev_windows":
        int(
            len(
                DEV_MC_STD_FD002_628FA
            )
        ),

    "test_endpoints":
        int(
            len(
                TEST_MC_STD_FD002_628FA
            )
        ),

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 18. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "DEV uncertainty values      :",
    len(
        DEV_MC_STD_FD002_628FA
    )
)

print(
    "TEST uncertainty values     :",
    len(
        TEST_MC_STD_FD002_628FA
    )
)

print(
    "MC passes                   :",
    MC_PASSES_628FA
)

print(
    "STD convention              : ddof=0"
)

print(
    "DEV deterministic reproduced:",
    bool(
        dev_reproduction_error_628fa
        <=
        1e-6
    )
)

print(
    "DEV MC-STD finite           :",
    bool(
        np.isfinite(
            DEV_MC_STD_FD002_628FA
        ).all()
    )
)

print(
    "TEST MC-STD finite          :",
    bool(
        np.isfinite(
            TEST_MC_STD_FD002_628FA
        ).all()
    )
)

print(
    "Official TEST labels read   : False"
)

print(
    "TEST scored                 : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Model retrained             : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28F-A ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-A-R1
# DETERMINISTIC DEV REPRODUCTION REPAIR
#
# Reason:
#   Step 6.28E-B created DEV predictions using batch size 256.
#   Step 6.28F-A accidentally checked them using batch size 512.
#
# No retraining.
# No TEST labels.
# No files written.
# ============================================================

import numpy as np
import torch

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A-R1 — DEV REPRODUCTION REPAIR\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Original deterministic DEV loader contract
# ------------------------------------------------------------

ORIGINAL_DEV_BATCH_628FAR1 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "validation_batch_size"
    ]
)


print(
    "Original DEV batch size :",
    ORIGINAL_DEV_BATCH_628FAR1
)


if ORIGINAL_DEV_BATCH_628FAR1 != 256:

    raise RuntimeError(
        "Unexpected frozen DEV batch size."
    )


dev_recheck_dataset_628far1 = (
    TensorDataset(

        torch.from_numpy(

            np.asarray(
                X_DEV_FD002_628D,
                dtype=np.float32,
            )
        )
    )
)


dev_recheck_loader_628far1 = (
    DataLoader(

        dev_recheck_dataset_628far1,

        batch_size=ORIGINAL_DEV_BATCH_628FAR1,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE_628FA.type
            ==
            "cuda"
        ),
    )
)


# ------------------------------------------------------------
# 2. Deterministic re-inference using original batch contract
# ------------------------------------------------------------

DEV_DETERMINISTIC_RECHECK_628FAR1 = (
    deterministic_predict_x_628fa(

        MODEL_FD002_628EB,

        dev_recheck_loader_628far1,
    )
)


stored_dev_628far1 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
)


difference_628far1 = (

    DEV_DETERMINISTIC_RECHECK_628FAR1

    -

    stored_dev_628far1
)


max_abs_error_628far1 = float(
    np.max(
        np.abs(
            difference_628far1
        )
    )
)


mean_abs_error_628far1 = float(
    np.mean(
        np.abs(
            difference_628far1
        )
    )
)


rmse_difference_628far1 = float(
    np.sqrt(
        np.mean(
            difference_628far1 ** 2
        )
    )
)


# Numerical-equivalence check appropriate for float32 GPU output
numerically_equivalent_628far1 = bool(
    np.allclose(

        DEV_DETERMINISTIC_RECHECK_628FAR1,

        stored_dev_628far1,

        rtol=1e-6,

        atol=1e-4,
    )
)


print(
    "\n--- REPRODUCTION RESULTS ---"
)

print(
    "Maximum absolute error :",
    max_abs_error_628far1
)

print(
    "Mean absolute error    :",
    mean_abs_error_628far1
)

print(
    "Difference RMSE        :",
    rmse_difference_628far1
)

print(
    "Numerically equivalent :",
    numerically_equivalent_628far1
)


if not numerically_equivalent_628far1:

    raise RuntimeError(
        "DEV predictions are not numerically equivalent "
        "under the original inference contract."
    )


# ------------------------------------------------------------
# 3. Freeze corrected reproduction evidence
# ------------------------------------------------------------

FD002_DEV_REPRODUCTION_628FAR1 = {

    "original_dev_batch_size":
        ORIGINAL_DEV_BATCH_628FAR1,

    "max_absolute_error":
        max_abs_error_628far1,

    "mean_absolute_error":
        mean_abs_error_628far1,

    "difference_rmse":
        rmse_difference_628far1,

    "rtol":
        1e-6,

    "atol":
        1e-4,

    "numerically_equivalent":
        numerically_equivalent_628far1,

    "model_retrained":
        False,

    "official_test_labels_read":
        False,

    "files_written":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Original inference batch     :",
    ORIGINAL_DEV_BATCH_628FAR1
)

print(
    "Maximum absolute difference  :",
    max_abs_error_628far1
)

print(
    "Numerically equivalent       :",
    numerically_equivalent_628far1
)

print(
    "Model retrained              : False"
)

print(
    "Official TEST labels read    : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28F-A-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-A-R2
# COMPLETE FD002 CLEAN MC-DROPOUT UNCERTAINTY
#
# This continues after the successful exact DEV
# reproduction repair.
#
# Deterministic inference:
#   batch = 256
#
# MC-dropout:
#   passes = 50
#   batch  = 512
#   seed   = 42
#   std    = ddof=0
#
# IMPORTANT:
#   - no model retraining
#   - official FD002 TEST labels NOT read
#   - TEST NOT scored
#   - no reliability threshold selected
#   - no files written
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A-R2 — COMPLETE CLEAN MC-DROPOUT UNCERTAINTY\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_628far2 = [
    "MODEL_FD002_628EB",
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "X_DEV_FD002_628D",
    "X_TEST_FD002_628D",
    "DEV_WINDOW_KEYS_FD002_628D",
    "TEST_ENDPOINT_META_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",
    "DEV_DETERMINISTIC_RECHECK_628FAR1",
    "FD002_DEV_REPRODUCTION_628FAR1",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_628far2:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


if not bool(
    FD002_DEV_REPRODUCTION_628FAR1[
        "numerically_equivalent"
    ]
):

    raise RuntimeError(
        "DEV reproduction repair was not validated."
    )


# ============================================================
# 2. Frozen inference contracts
# ============================================================

DETERMINISTIC_BATCH_628FAR2 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "validation_batch_size"
    ]
)

MC_BATCH_628FAR2 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_batch_size"
    ]
)

MC_PASSES_628FAR2 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_passes"
    ]
)

MC_SEED_628FAR2 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_seed"
    ]
)


print("\n--- FROZEN INFERENCE CONTRACT ---")

print(
    "Deterministic batch :",
    DETERMINISTIC_BATCH_628FAR2
)

print(
    "MC batch            :",
    MC_BATCH_628FAR2
)

print(
    "MC passes           :",
    MC_PASSES_628FAR2
)

print(
    "MC seed             :",
    MC_SEED_628FAR2
)

print(
    "MC STD ddof         : 0"
)


if DETERMINISTIC_BATCH_628FAR2 != 256:
    raise RuntimeError(
        "Expected deterministic batch = 256."
    )

if MC_BATCH_628FAR2 != 512:
    raise RuntimeError(
        "Expected MC batch = 512."
    )

if MC_PASSES_628FAR2 != 50:
    raise RuntimeError(
        "Expected MC passes = 50."
    )

if MC_SEED_628FAR2 != 42:
    raise RuntimeError(
        "Expected MC seed = 42."
    )


# ============================================================
# 3. Device + arrays
# ============================================================

DEVICE_628FAR2 = next(
    MODEL_FD002_628EB.parameters()
).device


X_dev_628far2 = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32,
)

X_test_628far2 = np.asarray(
    X_TEST_FD002_628D,
    dtype=np.float32,
)


print("\n--- INPUT CONTRACT ---")

print(
    "Device :",
    DEVICE_628FAR2
)

print(
    "DEV X  :",
    X_dev_628far2.shape
)

print(
    "TEST X :",
    X_test_628far2.shape
)


if X_dev_628far2.shape != (
    8367,
    30,
    17,
):
    raise RuntimeError(
        "Unexpected DEV shape."
    )


if X_test_628far2.shape != (
    259,
    30,
    17,
):
    raise RuntimeError(
        "Unexpected TEST shape."
    )


# ============================================================
# 4. Construct deterministic TEST loader
#
# Use the frozen deterministic inference batch = 256.
# ============================================================

test_det_dataset_628far2 = TensorDataset(
    torch.from_numpy(
        X_test_628far2
    )
)


test_det_loader_628far2 = DataLoader(
    test_det_dataset_628far2,
    batch_size=DETERMINISTIC_BATCH_628FAR2,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE_628FAR2.type
        ==
        "cuda"
    ),
)


# ============================================================
# 5. Deterministic helper
# ============================================================

@torch.inference_mode()
def deterministic_predict_628far2(
    model,
    loader,
):

    model.eval()

    outputs = []


    for (X_batch,) in loader:

        X_batch = X_batch.to(
            DEVICE_628FAR2,
            non_blocking=True,
        )


        pred = model(
            X_batch
        )


        outputs.append(
            pred.detach()
            .cpu()
            .numpy()
        )


    return (
        np.concatenate(
            outputs
        )
        .astype(
            np.float64
        )
    )


# ============================================================
# 6. Label-free deterministic TEST prediction
# ============================================================

TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2 = (
    deterministic_predict_628far2(
        MODEL_FD002_628EB,
        test_det_loader_628far2,
    )
)


if len(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
) != 259:

    raise RuntimeError(
        "TEST prediction count mismatch."
    )


if not np.isfinite(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
).all():

    raise RuntimeError(
        "Non-finite deterministic TEST predictions."
    )


print(
    "\n--- LABEL-FREE DETERMINISTIC TEST PREDICTIONS ---"
)

print(
    "Count   :",
    len(
        TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
    )
)

print(
    "Minimum :",
    float(
        np.min(
            TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
        )
    )
)

print(
    "Median  :",
    float(
        np.median(
            TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
        )
    )
)

print(
    "Mean    :",
    float(
        np.mean(
            TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
        )
    )
)

print(
    "Maximum :",
    float(
        np.max(
            TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
        )
    )
)


# ============================================================
# 7. MC loaders
# ============================================================

dev_mc_dataset_628far2 = TensorDataset(
    torch.from_numpy(
        X_dev_628far2
    )
)


test_mc_dataset_628far2 = TensorDataset(
    torch.from_numpy(
        X_test_628far2
    )
)


dev_mc_loader_628far2 = DataLoader(
    dev_mc_dataset_628far2,
    batch_size=MC_BATCH_628FAR2,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE_628FAR2.type
        ==
        "cuda"
    ),
)


test_mc_loader_628far2 = DataLoader(
    test_mc_dataset_628far2,
    batch_size=MC_BATCH_628FAR2,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE_628FAR2.type
        ==
        "cuda"
    ),
)


# ============================================================
# 8. MC-dropout helper
#
# model.train() activates the explicit Dropout(p=0.20).
#
# The frozen GRU itself has one layer, so there is no
# inter-layer GRU dropout.
# ============================================================

@torch.inference_mode()
def mc_dropout_predict_628far2(
    model,
    loader,
    n_samples,
    seed,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    n_observations = len(
        loader.dataset
    )


    mc_matrix = np.empty(
        (
            n_samples,
            n_observations,
        ),
        dtype=np.float32,
    )


    model.train()


    for sample_index in range(
        n_samples
    ):

        sample_predictions = []


        for (X_batch,) in loader:

            X_batch = X_batch.to(
                DEVICE_628FAR2,
                non_blocking=True,
            )


            pred = model(
                X_batch
            )


            sample_predictions.append(
                pred.detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


        sample_predictions = np.concatenate(
            sample_predictions
        )


        if len(
            sample_predictions
        ) != n_observations:

            raise RuntimeError(
                "MC sample prediction count mismatch."
            )


        mc_matrix[
            sample_index
        ] = sample_predictions


    model.eval()


    mc_mean = (
        mc_matrix
        .mean(
            axis=0
        )
        .astype(
            np.float64
        )
    )


    mc_std = (
        mc_matrix
        .std(
            axis=0,
            ddof=0,
        )
        .astype(
            np.float64
        )
    )


    return (
        mc_matrix,
        mc_mean,
        mc_std,
    )


# ============================================================
# 9. Clean DEV MC-dropout
# ============================================================

print(
    "\n--- MC-DROPOUT: CLEAN DEV ---"
)

print(
    f"Running {MC_PASSES_628FAR2} "
    "stochastic forward passes..."
)


(
    DEV_MC_MATRIX_FD002_628FAR2,
    DEV_MC_MEAN_FD002_628FAR2,
    DEV_MC_STD_FD002_628FAR2,

) = mc_dropout_predict_628far2(

    MODEL_FD002_628EB,

    dev_mc_loader_628far2,

    n_samples=MC_PASSES_628FAR2,

    seed=MC_SEED_628FAR2,
)


# ============================================================
# 10. Label-free TEST MC-dropout
# ============================================================

print(
    "\n--- MC-DROPOUT: LABEL-FREE TEST ---"
)

print(
    f"Running {MC_PASSES_628FAR2} "
    "stochastic forward passes..."
)


(
    TEST_MC_MATRIX_FD002_628FAR2,
    TEST_MC_MEAN_FD002_628FAR2,
    TEST_MC_STD_FD002_628FAR2,

) = mc_dropout_predict_628far2(

    MODEL_FD002_628EB,

    test_mc_loader_628far2,

    n_samples=MC_PASSES_628FAR2,

    seed=MC_SEED_628FAR2,
)


# ============================================================
# 11. Numerical-integrity checks
# ============================================================

integrity_628far2 = pd.DataFrame(
    [
        {
            "population":
                "DEV",

            "n":
                len(
                    DEV_MC_STD_FD002_628FAR2
                ),

            "mean_finite":
                bool(
                    np.isfinite(
                        DEV_MC_MEAN_FD002_628FAR2
                    ).all()
                ),

            "std_finite":
                bool(
                    np.isfinite(
                        DEV_MC_STD_FD002_628FAR2
                    ).all()
                ),

            "negative_std":
                int(
                    np.sum(
                        DEV_MC_STD_FD002_628FAR2
                        <
                        0
                    )
                ),
        },

        {
            "population":
                "TEST",

            "n":
                len(
                    TEST_MC_STD_FD002_628FAR2
                ),

            "mean_finite":
                bool(
                    np.isfinite(
                        TEST_MC_MEAN_FD002_628FAR2
                    ).all()
                ),

            "std_finite":
                bool(
                    np.isfinite(
                        TEST_MC_STD_FD002_628FAR2
                    ).all()
                ),

            "negative_std":
                int(
                    np.sum(
                        TEST_MC_STD_FD002_628FAR2
                        <
                        0
                    )
                ),
        },
    ]
)


print(
    "\n--- MC NUMERICAL INTEGRITY ---"
)

print(
    integrity_628far2.to_string(
        index=False
    )
)


if not bool(
    integrity_628far2[
        "mean_finite"
    ].all()
):

    raise RuntimeError(
        "Non-finite MC means."
    )


if not bool(
    integrity_628far2[
        "std_finite"
    ].all()
):

    raise RuntimeError(
        "Non-finite MC uncertainty."
    )


# ============================================================
# 12. Distribution helper
# ============================================================

def uncertainty_summary_628far2(
    values
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )


    return {
        "n":
            int(
                len(
                    values
                )
            ),

        "minimum":
            float(
                np.min(
                    values
                )
            ),

        "q25":
            float(
                np.quantile(
                    values,
                    0.25,
                )
            ),

        "median":
            float(
                np.median(
                    values
                )
            ),

        "q75":
            float(
                np.quantile(
                    values,
                    0.75,
                )
            ),

        "maximum":
            float(
                np.max(
                    values
                )
            ),

        "mean":
            float(
                np.mean(
                    values
                )
            ),
    }


DEV_MC_SUMMARY_FD002_628FAR2 = (
    uncertainty_summary_628far2(
        DEV_MC_STD_FD002_628FAR2
    )
)


TEST_MC_SUMMARY_FD002_628FAR2 = (
    uncertainty_summary_628far2(
        TEST_MC_STD_FD002_628FAR2
    )
)


print(
    "\n--- RAW MC-STD DISTRIBUTIONS ---"
)


print("\nDEV")

for key, value in (
    DEV_MC_SUMMARY_FD002_628FAR2.items()
):

    if key == "n":

        print(
            f"{key:<8}: {value}"
        )

    else:

        print(
            f"{key:<8}: {value:.9f}"
        )


print("\nTEST")

for key, value in (
    TEST_MC_SUMMARY_FD002_628FAR2.items()
):

    if key == "n":

        print(
            f"{key:<8}: {value}"
        )

    else:

        print(
            f"{key:<8}: {value:.9f}"
        )


# ============================================================
# 13. MC mean vs deterministic prediction
#
# Descriptive diagnostic only.
# ============================================================

dev_mcmean_vs_det_mae_628far2 = float(
    np.mean(
        np.abs(
            DEV_MC_MEAN_FD002_628FAR2
            -
            np.asarray(
                DEV_DETERMINISTIC_RECHECK_628FAR1,
                dtype=np.float64,
            )
        )
    )
)


test_mcmean_vs_det_mae_628far2 = float(
    np.mean(
        np.abs(
            TEST_MC_MEAN_FD002_628FAR2
            -
            TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
        )
    )
)


print(
    "\n--- MC MEAN VS DETERMINISTIC ---"
)

print(
    "DEV mean absolute difference  :",
    dev_mcmean_vs_det_mae_628far2
)

print(
    "TEST mean absolute difference :",
    test_mcmean_vs_det_mae_628far2
)


# ============================================================
# 14. Canonical DEV uncertainty table
# ============================================================

if len(
    DEV_WINDOW_KEYS_FD002_628D
) != len(
    DEV_MC_STD_FD002_628FAR2
):

    raise RuntimeError(
        "DEV key alignment failure."
    )


DEV_CLEAN_UNCERTAINTY_FD002_628FAR2 = (
    DEV_WINDOW_KEYS_FD002_628D.copy()
)


DEV_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "deterministic_prediction"
] = np.asarray(
    DEV_DETERMINISTIC_RECHECK_628FAR1,
    dtype=np.float64,
)


DEV_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "mc_mean"
] = DEV_MC_MEAN_FD002_628FAR2


DEV_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "mc_std"
] = DEV_MC_STD_FD002_628FAR2


# ============================================================
# 15. Canonical label-free TEST table
# ============================================================

TEST_CLEAN_UNCERTAINTY_FD002_628FAR2 = (
    TEST_ENDPOINT_META_FD002_628D.copy()
)


if len(
    TEST_CLEAN_UNCERTAINTY_FD002_628FAR2
) != 259:

    raise RuntimeError(
        "Unexpected TEST metadata count."
    )


TEST_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "deterministic_prediction"
] = (
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
)


TEST_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "mc_mean"
] = (
    TEST_MC_MEAN_FD002_628FAR2
)


TEST_CLEAN_UNCERTAINTY_FD002_628FAR2[
    "mc_std"
] = (
    TEST_MC_STD_FD002_628FAR2
)


# ============================================================
# 16. Padded-vs-natural TEST uncertainty
#
# LABEL-FREE descriptive diagnostic only.
# ============================================================

test_padding_summary_628far2 = (

    TEST_CLEAN_UNCERTAINTY_FD002_628FAR2

    .groupby(
        "padding_used"
    )

    .agg(
        engines=(
            "unit_id",
            "count",
        ),

        mean_mc_std=(
            "mc_std",
            "mean",
        ),

        median_mc_std=(
            "mc_std",
            "median",
        ),
    )

    .reset_index()
)


print(
    "\n--- TEST PADDING / UNCERTAINTY DIAGNOSTIC ---"
)

print(
    test_padding_summary_628far2
    .to_string(
        index=False
    )
)


# ============================================================
# 17. Freeze completed uncertainty contract
# ============================================================

FD002_MC_CONTRACT_628FAR2 = {

    "dataset":
        "FD002",

    "deterministic_batch_size":
        DETERMINISTIC_BATCH_628FAR2,

    "mc_batch_size":
        MC_BATCH_628FAR2,

    "mc_passes":
        MC_PASSES_628FAR2,

    "mc_seed":
        MC_SEED_628FAR2,

    "std_ddof":
        0,

    "dev_windows":
        int(
            len(
                DEV_MC_STD_FD002_628FAR2
            )
        ),

    "test_endpoints":
        int(
            len(
                TEST_MC_STD_FD002_628FAR2
            )
        ),

    "dev_reproduction_exact":
        True,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 18. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-A-R2 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "DEV uncertainty values       :",
    len(
        DEV_MC_STD_FD002_628FAR2
    )
)

print(
    "TEST uncertainty values      :",
    len(
        TEST_MC_STD_FD002_628FAR2
    )
)

print(
    "MC passes                    :",
    MC_PASSES_628FAR2
)

print(
    "MC batch                     :",
    MC_BATCH_628FAR2
)

print(
    "STD convention               : ddof=0"
)

print(
    "DEV reproduction exact       : True"
)

print(
    "DEV MC uncertainty finite    :",
    bool(
        np.isfinite(
            DEV_MC_STD_FD002_628FAR2
        ).all()
    )
)

print(
    "TEST MC uncertainty finite   :",
    bool(
        np.isfinite(
            TEST_MC_STD_FD002_628FAR2
        ).all()
    )
)

print(
    "Official TEST labels read    : False"
)

print(
    "TEST scored                  : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Model retrained              : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28F-A-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-B0
# RECOVER EXACT FD001 UNCERTAINTY-TRANSFORMATION CONTRACT
#
# PURPOSE
# -------
# Extract only the relevant source cells from the completed
# FD001 reliability notebook so that the FD002 replication
# uses the EXACT same uncertainty transformation.
#
# NO notebook execution.
# NO model training.
# NO TEST labels.
# NO files written.
# ============================================================

from pathlib import Path
import json


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0 — FD001 UNCERTAINTY CONTRACT RECOVERY\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_628fb0 = Path.cwd()


if (
    cwd_628fb0
    /
    "03_notebooks"
).exists():

    PROJECT_ROOT_628FB0 = (
        cwd_628fb0
    )


elif (
    cwd_628fb0.parent
    /
    "03_notebooks"
).exists():

    PROJECT_ROOT_628FB0 = (
        cwd_628fb0.parent
    )


else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


fd001_notebook_628fb0 = (

    PROJECT_ROOT_628FB0

    /

    "03_notebooks"

    /

    "06_reliability_signals_fd001.ipynb"
)


print("\n--- NOTEBOOK SOURCE ---")

print(
    "Project root :",
    PROJECT_ROOT_628FB0
)

print(
    "Notebook     :",
    fd001_notebook_628fb0
)

print(
    "Exists       :",
    fd001_notebook_628fb0.exists()
)


if not fd001_notebook_628fb0.exists():

    raise FileNotFoundError(
        fd001_notebook_628fb0
    )


# ------------------------------------------------------------
# 2. Read notebook as JSON only
# ------------------------------------------------------------

with open(
    fd001_notebook_628fb0,
    "r",
    encoding="utf-8",
) as f:

    nb_628fb0 = json.load(
        f
    )


cells_628fb0 = nb_628fb0.get(
    "cells",
    []
)


print(
    "Notebook cells:",
    len(
        cells_628fb0
    )
)


# ------------------------------------------------------------
# 3. Targeted search terms
#
# Deliberately narrow so VS Code does not produce thousands
# of irrelevant lines.
# ------------------------------------------------------------

search_terms_628fb0 = [

    "phase_adjusted_uncertainty",

    "phase-adjusted",

    "phase adjustment",

    "right_ecdf",

    "midrank_ecdf",

    "clean_adjusted_uncertainty",

    "corrupted_adjusted_uncertainty",

    "clean_uncertainty_risk",

    "corrupted_uncertainty_risk",

    "FROZEN_VAL_UNCERTAINTY",

    "UNCERTAINTY_RISK_REFERENCE",

    "RELIABILITY_UNCERTAINTY_REFERENCE",

    "u_reference",

]


# ------------------------------------------------------------
# 4. Find relevant cells
# ------------------------------------------------------------

matches_628fb0 = []


for cell_index_628fb0, cell_628fb0 in enumerate(
    cells_628fb0
):

    source_628fb0 = "".join(
        cell_628fb0.get(
            "source",
            []
        )
    )


    source_lower_628fb0 = (
        source_628fb0.lower()
    )


    matched_terms_628fb0 = [

        term_628fb0

        for term_628fb0
        in search_terms_628fb0

        if (
            term_628fb0.lower()
            in
            source_lower_628fb0
        )
    ]


    if matched_terms_628fb0:

        matches_628fb0.append(
            {
                "cell_index":
                    cell_index_628fb0,

                "cell_type":
                    cell_628fb0.get(
                        "cell_type",
                        "unknown",
                    ),

                "terms":
                    matched_terms_628fb0,

                "source":
                    source_628fb0,
            }
        )


print(
    "\nRelevant cells found:",
    len(
        matches_628fb0
    )
)


# ------------------------------------------------------------
# 5. Print compact source
#
# Maximum 12 matching cells to keep output manageable.
# ------------------------------------------------------------

MAX_CELLS_628FB0 = 12


for match_number_628fb0, match_628fb0 in enumerate(
    matches_628fb0[
        :MAX_CELLS_628FB0
    ],
    start=1,
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"MATCH {match_number_628fb0}"
    )

    print(
        "Notebook cell index :",
        match_628fb0[
            "cell_index"
        ]
    )

    print(
        "Cell type           :",
        match_628fb0[
            "cell_type"
        ]
    )

    print(
        "Matched terms       :",
        match_628fb0[
            "terms"
        ]
    )

    print(
        "-" * 70
    )


    source_lines_628fb0 = (
        match_628fb0[
            "source"
        ]
        .splitlines()
    )


    # Print at most 180 source lines from each relevant cell.
    for line_number_628fb0, line_628fb0 in enumerate(
        source_lines_628fb0[
            :180
        ],
        start=1,
    ):

        print(
            f"{line_number_628fb0:03d}: "
            f"{line_628fb0}"
        )


    if len(
        source_lines_628fb0
    ) > 180:

        print(
            "... SOURCE TRUNCATED AFTER 180 LINES ..."
        )


# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 notebook executed       : False"
)

print(
    "Relevant source cells found   :",
    len(
        matches_628fb0
    )
)

print(
    "Cells displayed               :",
    min(
        len(
            matches_628fb0
        ),
        MAX_CELLS_628FB0,
    )
)

print(
    "FD002 model retrained         : False"
)

print(
    "Official FD002 TEST labels read: False"
)

print(
    "Threshold selected            : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.28F-B0 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-B0-R
# TARGETED FD001 UNCERTAINTY RECIPE RECOVERY
#
# PURPOSE
# -------
# Recover ONLY the exact missing implementation required for
# faithful FD002 replication:
#
#   1. phase_adjusted_uncertainty_623cb2
#   2. creation/use of phase_references_622h
#   3. det_pred_pct_b5 transformation
#
# NO notebook execution.
# NO model training.
# NO TEST labels.
# NO files written.
# ============================================================

from pathlib import Path
import json


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0-R — TARGETED FD001 RECIPE RECOVERY\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_628fb0r = Path.cwd()

if (cwd_628fb0r / "03_notebooks").exists():
    PROJECT_ROOT_628FB0R = cwd_628fb0r

elif (cwd_628fb0r.parent / "03_notebooks").exists():
    PROJECT_ROOT_628FB0R = cwd_628fb0r.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


notebook_path_628fb0r = (
    PROJECT_ROOT_628FB0R
    / "03_notebooks"
    / "06_reliability_signals_fd001.ipynb"
)


if not notebook_path_628fb0r.exists():
    raise FileNotFoundError(
        notebook_path_628fb0r
    )


with open(
    notebook_path_628fb0r,
    "r",
    encoding="utf-8",
) as f:
    nb_628fb0r = json.load(f)


cells_628fb0r = nb_628fb0r["cells"]


print(
    "Notebook:",
    notebook_path_628fb0r
)

print(
    "Cells:",
    len(cells_628fb0r)
)


# ------------------------------------------------------------
# 2. Exact target strings
# ------------------------------------------------------------

targets_628fb0r = [

    "def phase_adjusted_uncertainty_623cb2",

    "phase_references_622h",

    '"det_pred_pct_b5"',

    "det_pred_pct_b5",

]


# ------------------------------------------------------------
# 3. Print context around EVERY exact occurrence
# ------------------------------------------------------------

hits_628fb0r = 0


for cell_idx_628fb0r, cell_628fb0r in enumerate(
    cells_628fb0r
):

    if cell_628fb0r.get("cell_type") != "code":
        continue

    source_628fb0r = "".join(
        cell_628fb0r.get(
            "source",
            []
        )
    )

    lines_628fb0r = (
        source_628fb0r.splitlines()
    )


    matching_line_indices_628fb0r = []


    for line_idx_628fb0r, line_628fb0r in enumerate(
        lines_628fb0r
    ):

        if any(
            target_628fb0r in line_628fb0r
            for target_628fb0r
            in targets_628fb0r
        ):

            matching_line_indices_628fb0r.append(
                line_idx_628fb0r
            )


    if not matching_line_indices_628fb0r:
        continue


    hits_628fb0r += 1


    # Merge into one useful context window for this cell.
    start_628fb0r = max(
        0,
        min(
            matching_line_indices_628fb0r
        ) - 80
    )

    stop_628fb0r = min(
        len(lines_628fb0r),
        max(
            matching_line_indices_628fb0r
        ) + 181
    )


    print(
        "\n"
        + "=" * 78
    )

    print(
        "CELL INDEX:",
        cell_idx_628fb0r
    )

    print(
        "TOTAL CELL LINES:",
        len(lines_628fb0r)
    )

    print(
        "MATCHING SOURCE LINE(S):",
        [
            x + 1
            for x in
            matching_line_indices_628fb0r
        ]
    )

    print(
        "PRINTED SOURCE RANGE:",
        start_628fb0r + 1,
        "to",
        stop_628fb0r
    )

    print(
        "-" * 78
    )


    for i_628fb0r in range(
        start_628fb0r,
        stop_628fb0r
    ):

        print(
            f"{i_628fb0r + 1:04d}: "
            f"{lines_628fb0r[i_628fb0r]}"
        )


# ------------------------------------------------------------
# 4. Special full extraction of cell 141
#
# Our previous recovery showed that this cell contains the
# exact end-to-end validation reliability implementation.
# ------------------------------------------------------------

CELL_141_628FB0R = 141


if CELL_141_628FB0R < len(
    cells_628fb0r
):

    cell141_source_628fb0r = "".join(
        cells_628fb0r[
            CELL_141_628FB0R
        ].get(
            "source",
            []
        )
    )

    cell141_lines_628fb0r = (
        cell141_source_628fb0r.splitlines()
    )


    print(
        "\n"
        + "=" * 78
    )

    print(
        "SPECIAL EXTRACTION — CELL 141"
    )

    print(
        "=" * 78
    )


    # Print from line 170 onward because the previous
    # extractor stopped at line 180.
    for i_628fb0r in range(
        160,
        len(
            cell141_lines_628fb0r
        )
    ):

        print(
            f"{i_628fb0r + 1:04d}: "
            f"{cell141_lines_628fb0r[i_628fb0r]}"
        )


# ------------------------------------------------------------
# 5. Final safety summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0-R FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Relevant source cells found :",
    hits_628fb0r
)

print(
    "FD001 notebook executed     : False"
)

print(
    "FD002 model retrained       : False"
)

print(
    "Official FD002 TEST labels read : False"
)

print(
    "TEST scored                 : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28F-B0-R ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-B0-R2
# RECOVER EXACT FD001 PHASE-REFERENCE FITTING HELPERS
#
# INSPECTION ONLY
# NO NOTEBOOK EXECUTION
# NO MODEL TRAINING
# NO TEST LABELS
# NO FILE WRITING
# ============================================================

from pathlib import Path
import json

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0-R2 — EXACT PHASE-REFERENCE HELPER RECOVERY\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_628fb0r2 = Path.cwd()

if (cwd_628fb0r2 / "03_notebooks").exists():
    PROJECT_ROOT_628FB0R2 = cwd_628fb0r2

elif (cwd_628fb0r2.parent / "03_notebooks").exists():
    PROJECT_ROOT_628FB0R2 = cwd_628fb0r2.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


fd001_nb_path_628fb0r2 = (
    PROJECT_ROOT_628FB0R2
    / "03_notebooks"
    / "06_reliability_signals_fd001.ipynb"
)

if not fd001_nb_path_628fb0r2.exists():
    raise FileNotFoundError(
        fd001_nb_path_628fb0r2
    )


with open(
    fd001_nb_path_628fb0r2,
    "r",
    encoding="utf-8"
) as f_628fb0r2:

    nb_628fb0r2 = json.load(
        f_628fb0r2
    )


# ------------------------------------------------------------
# 2. Exact helper definitions required
# ------------------------------------------------------------

TARGET_DEFS_628FB0R2 = [

    "def fit_phase_reference_622h",

    "def apply_phase_percentile_622h",

]


found_628fb0r2 = []


# ------------------------------------------------------------
# 3. Search every FD001 code cell
# ------------------------------------------------------------

for cell_idx_628fb0r2, cell_628fb0r2 in enumerate(
    nb_628fb0r2["cells"]
):

    if cell_628fb0r2.get(
        "cell_type"
    ) != "code":
        continue


    source_628fb0r2 = "".join(
        cell_628fb0r2.get(
            "source",
            []
        )
    )


    matched_defs_628fb0r2 = [
        target_628fb0r2
        for target_628fb0r2
        in TARGET_DEFS_628FB0R2
        if target_628fb0r2
        in source_628fb0r2
    ]


    if not matched_defs_628fb0r2:
        continue


    found_628fb0r2.append(
        (
            cell_idx_628fb0r2,
            matched_defs_628fb0r2
        )
    )


    lines_628fb0r2 = (
        source_628fb0r2.splitlines()
    )


    print(
        "\n"
        + "=" * 78
    )

    print(
        "NOTEBOOK CELL INDEX :",
        cell_idx_628fb0r2
    )

    print(
        "MATCHED DEFINITIONS :",
        matched_defs_628fb0r2
    )

    print(
        "TOTAL CELL LINES    :",
        len(lines_628fb0r2)
    )

    print(
        "-" * 78
    )


    # Print complete cell.
    # This guarantees that helper logic, quantile rules,
    # bin-edge treatment, and reference construction are visible.

    for line_no_628fb0r2, line_628fb0r2 in enumerate(
        lines_628fb0r2,
        start=1
    ):

        print(
            f"{line_no_628fb0r2:04d}: "
            f"{line_628fb0r2}"
        )


# ------------------------------------------------------------
# 4. Safety / completeness check
# ------------------------------------------------------------

located_names_628fb0r2 = set()

for _, matches_628fb0r2 in found_628fb0r2:

    located_names_628fb0r2.update(
        matches_628fb0r2
    )


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B0-R2 FINAL SUMMARY\n"
    "============================================================"
)

for target_628fb0r2 in TARGET_DEFS_628FB0R2:

    print(
        f"{target_628fb0r2:<40}:",
        target_628fb0r2
        in located_names_628fb0r2
    )


print(
    "\nMatching source cells :",
    len(
        found_628fb0r2
    )
)

print(
    "FD001 notebook executed       : False"
)

print(
    "FD002 model retrained         : False"
)

print(
    "Official FD002 TEST labels read:",
    False
)

print(
    "TEST scored                   :",
    False
)

print(
    "Threshold selected            :",
    False
)

print(
    "Files written                 :",
    False
)

print(
    "\n=== END STEP 6.28F-B0-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-B
# FD002 DEVELOPMENT-CALIBRATED PHASE-ADJUSTED UNCERTAINTY
#
# Frozen transferred recipe from FD001:
#     det_pred_pct_b5
#
# Recipe:
#     deterministic predicted RUL
#         -> 5 DEVELOPMENT quantile phase bins
#
#     raw MC-dropout std
#         -> empirical within-phase mid-rank percentile
#
# IMPORTANT:
# - Recipe STRUCTURE transferred from FD001
# - Numerical references fitted from FD002 DEV only
# - Official FD002 TEST labels are NOT read
# - No TEST scoring
# - No threshold selection
# - No abstention
# - No model retraining
# - No files written
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B — FD002 PHASE-ADJUSTED UNCERTAINTY\n"
    "============================================================"
)


# ============================================================
# 1. Core required FD002 objects
# ============================================================

core_required_628fb = [
    "DEV_PREDICTIONS_FD002_628EB",
    "X_DEV_FD002_628D",
    "X_TEST_FD002_628D",
]

print("\n--- CORE REQUIRED OBJECTS ---")

for name_628fb in core_required_628fb:
    print(
        f"{name_628fb:<38}:",
        name_628fb in globals()
    )

missing_core_628fb = [
    name_628fb
    for name_628fb in core_required_628fb
    if name_628fb not in globals()
]

if missing_core_628fb:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing_core_628fb)
    )


EXPECTED_DEV_628FB = int(
    X_DEV_FD002_628D.shape[0]
)

EXPECTED_TEST_628FB = int(
    X_TEST_FD002_628D.shape[0]
)

print(
    "\nExpected DEV windows :",
    EXPECTED_DEV_628FB
)

print(
    "Expected TEST engines:",
    EXPECTED_TEST_628FB
)


# ============================================================
# 2. Robust one-dimensional conversion
# ============================================================

def vec1d_628fb(obj):

    if hasattr(obj, "detach"):
        obj = (
            obj
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        obj,
        dtype=np.float64
    ).reshape(-1)


# ============================================================
# 3. Frozen DEV deterministic prediction
# ============================================================

dev_prediction_628fb = vec1d_628fb(
    DEV_PREDICTIONS_FD002_628EB
)

if len(dev_prediction_628fb) != EXPECTED_DEV_628FB:
    raise RuntimeError(
        "Unexpected DEV deterministic-prediction length: "
        f"{len(dev_prediction_628fb)}"
    )

if not np.isfinite(
    dev_prediction_628fb
).all():
    raise RuntimeError(
        "DEV deterministic predictions contain "
        "non-finite values."
    )


# ============================================================
# 4. Locate already-computed MC-STD vectors from Step 6.28F-A
#
# We deliberately reuse the previous clean MC-dropout results.
# No stochastic inference is rerun here.
# ============================================================

def mc_std_candidates_628fb(
    expected_length,
    split_token
):

    rows_628fb = []

    for name_628fb, obj_628fb in list(
        globals().items()
    ):

        lname_628fb = str(
            name_628fb
        ).lower()

        # Require clear MC-STD identity.
        if "std" not in lname_628fb:
            continue

        if "mc" not in lname_628fb:
            continue

        if split_token.lower() not in lname_628fb:
            continue

        try:
            arr_628fb = vec1d_628fb(
                obj_628fb
            )
        except Exception:
            continue

        if len(arr_628fb) != expected_length:
            continue

        if not np.isfinite(
            arr_628fb
        ).all():
            continue

        score_628fb = 0

        if "fd002" in lname_628fb:
            score_628fb += 10

        if "628f" in lname_628fb:
            score_628fb += 5

        if "dropout" in lname_628fb:
            score_628fb += 2

        if "raw" in lname_628fb:
            score_628fb += 1

        rows_628fb.append(
            {
                "name": name_628fb,
                "score": score_628fb,
                "array": arr_628fb,
            }
        )

    return rows_628fb


dev_std_candidates_628fb = (
    mc_std_candidates_628fb(
        EXPECTED_DEV_628FB,
        "dev"
    )
)

test_std_candidates_628fb = (
    mc_std_candidates_628fb(
        EXPECTED_TEST_628FB,
        "test"
    )
)


print(
    "\n--- EXISTING MC-STD CANDIDATES ---"
)

print("\nDEV:")
for row_628fb in sorted(
    dev_std_candidates_628fb,
    key=lambda x: x["score"],
    reverse=True
):
    print(
        f"{row_628fb['name']:<45} "
        f"score={row_628fb['score']}"
    )


print("\nTEST:")
for row_628fb in sorted(
    test_std_candidates_628fb,
    key=lambda x: x["score"],
    reverse=True
):
    print(
        f"{row_628fb['name']:<45} "
        f"score={row_628fb['score']}"
    )


if len(dev_std_candidates_628fb) == 0:
    raise RuntimeError(
        "No eligible FD002 DEV MC-STD vector was found."
    )

if len(test_std_candidates_628fb) == 0:
    raise RuntimeError(
        "No eligible FD002 TEST MC-STD vector was found."
    )


# ============================================================
# 5. Safely resolve MC-STD candidate
#
# If multiple highest-ranked aliases exist, they must be
# numerically identical. Otherwise stop rather than guessing.
# ============================================================

def resolve_candidate_628fb(
    candidates,
    label
):

    max_score_628fb = max(
        row_628fb["score"]
        for row_628fb
        in candidates
    )

    best_628fb = [
        row_628fb
        for row_628fb
        in candidates
        if row_628fb["score"]
        == max_score_628fb
    ]

    reference_628fb = (
        best_628fb[0]["array"]
    )

    for row_628fb in best_628fb[1:]:

        if not np.allclose(
            reference_628fb,
            row_628fb["array"],
            rtol=0.0,
            atol=1e-12
        ):

            raise RuntimeError(
                f"Ambiguous {label} MC-STD candidates "
                "with different numerical values."
            )

    print(
        f"\nResolved {label} MC-STD object :",
        best_628fb[0]["name"]
    )

    if len(best_628fb) > 1:

        print(
            f"{label} equivalent aliases       :",
            [
                row_628fb["name"]
                for row_628fb
                in best_628fb
            ]
        )

    return (
        reference_628fb.copy(),
        best_628fb[0]["name"]
    )


dev_mc_std_628fb, dev_std_name_628fb = (
    resolve_candidate_628fb(
        dev_std_candidates_628fb,
        "DEV"
    )
)

test_mc_std_628fb, test_std_name_628fb = (
    resolve_candidate_628fb(
        test_std_candidates_628fb,
        "TEST"
    )
)


# ============================================================
# 6. Locate existing deterministic TEST prediction
# ============================================================

def test_prediction_candidates_628fb():

    rows_628fb = []

    for name_628fb, obj_628fb in list(
        globals().items()
    ):

        lname_628fb = str(
            name_628fb
        ).lower()

        if "test" not in lname_628fb:
            continue

        if "pred" not in lname_628fb:
            continue

        # Reject stochastic/uncertainty objects.
        forbidden_628fb = [
            "std",
            "uncert",
            "error",
            "target",
            "rul_raw",
            "rul_cap",
        ]

        if any(
            token_628fb in lname_628fb
            for token_628fb
            in forbidden_628fb
        ):
            continue

        # MC mean is not the frozen phase variable.
        if "mc_mean" in lname_628fb:
            continue

        if "mcmean" in lname_628fb:
            continue

        try:
            arr_628fb = vec1d_628fb(
                obj_628fb
            )
        except Exception:
            continue

        if len(arr_628fb) != EXPECTED_TEST_628FB:
            continue

        if not np.isfinite(
            arr_628fb
        ).all():
            continue

        score_628fb = 0

        if "fd002" in lname_628fb:
            score_628fb += 10

        if "deterministic" in lname_628fb:
            score_628fb += 8

        if "628f" in lname_628fb:
            score_628fb += 5

        if "628e" in lname_628fb:
            score_628fb += 2

        rows_628fb.append(
            {
                "name": name_628fb,
                "score": score_628fb,
                "array": arr_628fb,
            }
        )

    return rows_628fb


test_pred_candidates_628fb = (
    test_prediction_candidates_628fb()
)

print(
    "\n--- TEST DETERMINISTIC-PREDICTION CANDIDATES ---"
)

for row_628fb in sorted(
    test_pred_candidates_628fb,
    key=lambda x: x["score"],
    reverse=True
):

    print(
        f"{row_628fb['name']:<50} "
        f"score={row_628fb['score']}"
    )


if len(test_pred_candidates_628fb) == 0:

    raise RuntimeError(
        "No existing label-free deterministic "
        "FD002 TEST prediction vector was found."
    )


test_prediction_628fb, test_pred_name_628fb = (
    resolve_candidate_628fb(
        test_pred_candidates_628fb,
        "TEST deterministic prediction"
    )
)


# ============================================================
# 7. Exact transferred FD001 fitting rule
#
# 5 quantile bins fitted using FD002 DEVELOPMENT
# deterministic predictions ONLY.
# ============================================================

FD002_REQUESTED_PHASE_BINS_628FB = 5

quantiles_628fb = np.linspace(
    0.0,
    1.0,
    FD002_REQUESTED_PHASE_BINS_628FB + 1
)

raw_edges_628fb = np.quantile(
    dev_prediction_628fb,
    quantiles_628fb
)

# Exact original FD001 rule:
# interior edges only + unique protection for ties.
FD002_PHASE_EDGES_628FB = np.unique(
    raw_edges_628fb[1:-1]
)


dev_phase_ids_628fb = np.digitize(
    dev_prediction_628fb,
    FD002_PHASE_EDGES_628FB,
    right=True
)


FD002_PHASE_REFERENCES_628FB = {}

for b_628fb in np.unique(
    dev_phase_ids_628fb
):

    values_628fb = (
        dev_mc_std_628fb[
            dev_phase_ids_628fb
            == b_628fb
        ]
    )

    FD002_PHASE_REFERENCES_628FB[
        int(b_628fb)
    ] = np.sort(
        values_628fb
    )


FD002_ACTUAL_PHASE_BINS_628FB = len(
    FD002_PHASE_REFERENCES_628FB
)


# ============================================================
# 8. Exact empirical mid-rank transformation
# ============================================================

def apply_fd002_phase_uncertainty_628fb(
    deterministic_prediction,
    mc_std_values,
    edges=FD002_PHASE_EDGES_628FB,
    references=FD002_PHASE_REFERENCES_628FB,
):

    phase_628fb = vec1d_628fb(
        deterministic_prediction
    )

    uncertainty_628fb = vec1d_628fb(
        mc_std_values
    )

    if phase_628fb.shape != uncertainty_628fb.shape:
        raise ValueError(
            "Prediction and MC-STD arrays must "
            "have identical shapes."
        )

    if not np.isfinite(
        phase_628fb
    ).all():

        raise ValueError(
            "Prediction array contains non-finite values."
        )

    if not np.isfinite(
        uncertainty_628fb
    ).all():

        raise ValueError(
            "MC-STD array contains non-finite values."
        )

    bin_ids_628fb = np.digitize(
        phase_628fb,
        edges,
        right=True
    )

    result_628fb = np.zeros(
        len(phase_628fb),
        dtype=np.float64
    )

    for b_628fb in np.unique(
        bin_ids_628fb
    ):

        mask_628fb = (
            bin_ids_628fb
            == b_628fb
        )

        if int(b_628fb) not in references:

            raise RuntimeError(
                "No FD002 DEVELOPMENT uncertainty "
                f"reference for phase bin {b_628fb}."
            )

        ref_628fb = np.asarray(
            references[
                int(b_628fb)
            ],
            dtype=np.float64
        )

        x_628fb = (
            uncertainty_628fb[
                mask_628fb
            ]
        )

        left_628fb = np.searchsorted(
            ref_628fb,
            x_628fb,
            side="left"
        )

        right_628fb = np.searchsorted(
            ref_628fb,
            x_628fb,
            side="right"
        )

        result_628fb[
            mask_628fb
        ] = (
            left_628fb
            + right_628fb
        ) / (
            2.0
            * len(ref_628fb)
        )

    return result_628fb


# ============================================================
# 9. Apply frozen recipe to DEV and label-free TEST
# ============================================================

FD002_DEV_ADJUSTED_UNCERTAINTY_628FB = (
    apply_fd002_phase_uncertainty_628fb(
        dev_prediction_628fb,
        dev_mc_std_628fb
    )
)


FD002_TEST_ADJUSTED_UNCERTAINTY_628FB = (
    apply_fd002_phase_uncertainty_628fb(
        test_prediction_628fb,
        test_mc_std_628fb
    )
)


# ============================================================
# 10. Structural / numerical audit
# ============================================================

def uncertainty_summary_628fb(
    name,
    values
):

    values = vec1d_628fb(
        values
    )

    return {
        "population": name,
        "n": len(values),
        "minimum": float(
            np.min(values)
        ),
        "q25": float(
            np.quantile(values, 0.25)
        ),
        "median": float(
            np.median(values)
        ),
        "mean": float(
            np.mean(values)
        ),
        "q75": float(
            np.quantile(values, 0.75)
        ),
        "p95": float(
            np.quantile(values, 0.95)
        ),
        "maximum": float(
            np.max(values)
        ),
        "finite": bool(
            np.isfinite(values).all()
        ),
    }


summary_628fb = pd.DataFrame(
    [
        uncertainty_summary_628fb(
            "DEV",
            FD002_DEV_ADJUSTED_UNCERTAINTY_628FB
        ),
        uncertainty_summary_628fb(
            "TEST",
            FD002_TEST_ADJUSTED_UNCERTAINTY_628FB
        ),
    ]
)


print(
    "\n--- FD002 FITTED PHASE REFERENCE ---"
)

print(
    "Transferred recipe      : det_pred_pct_b5"
)

print(
    "Requested phase bins    :",
    FD002_REQUESTED_PHASE_BINS_628FB
)

print(
    "Actual phase bins       :",
    FD002_ACTUAL_PHASE_BINS_628FB
)

print(
    "Interior phase edges    :",
    FD002_PHASE_EDGES_628FB.tolist()
)


print(
    "\n--- DEVELOPMENT REFERENCE BIN SIZES ---"
)

for b_628fb in sorted(
    FD002_PHASE_REFERENCES_628FB
):

    print(
        f"Phase {b_628fb}:",
        len(
            FD002_PHASE_REFERENCES_628FB[
                b_628fb
            ]
        )
    )


print(
    "\n--- PHASE-ADJUSTED UNCERTAINTY DISTRIBUTIONS ---"
)

print(
    summary_628fb
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Integrity conditions
# ============================================================

dev_adjusted_628fb = (
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FB
)

test_adjusted_628fb = (
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FB
)


dev_range_ok_628fb = bool(
    np.all(
        (dev_adjusted_628fb >= 0.0)
        &
        (dev_adjusted_628fb <= 1.0)
    )
)

test_range_ok_628fb = bool(
    np.all(
        (test_adjusted_628fb >= 0.0)
        &
        (test_adjusted_628fb <= 1.0)
    )
)


if not np.isfinite(
    dev_adjusted_628fb
).all():

    raise RuntimeError(
        "DEV adjusted uncertainty contains "
        "non-finite values."
    )


if not np.isfinite(
    test_adjusted_628fb
).all():

    raise RuntimeError(
        "TEST adjusted uncertainty contains "
        "non-finite values."
    )


if not dev_range_ok_628fb:

    raise RuntimeError(
        "DEV adjusted uncertainty is outside [0, 1]."
    )


if not test_range_ok_628fb:

    raise RuntimeError(
        "TEST adjusted uncertainty is outside [0, 1]."
    )


# ============================================================
# 12. Freeze in-memory methodology contract
# ============================================================

FD002_UNCERTAINTY_RECIPE_628FB = {

    "dataset":
        "FD002",

    "candidate":
        "det_pred_pct_b5",

    "selection_origin":
        "recipe_structure_transferred_from_FD001",

    "phase_source":
        "deterministic_RUL_prediction",

    "requested_phase_bins":
        5,

    "actual_phase_bins":
        FD002_ACTUAL_PHASE_BINS_628FB,

    "phase_edges":
        FD002_PHASE_EDGES_628FB.copy(),

    "phase_reference_source":
        "FD002_DEVELOPMENT_only",

    "phase_assignment":
        "numpy.digitize(prediction, edges, right=True)",

    "base_uncertainty":
        "MC_dropout_prediction_standard_deviation",

    "mc_std_convention":
        "population_standard_deviation_ddof_0",

    "within_phase_transformation":
        (
            "empirical_midrank_percentile:"
            "(count(ref<x)+count(ref<=x))/(2*N)"
        ),

    "dev_prediction_object":
        "DEV_PREDICTIONS_FD002_628EB",

    "dev_mc_std_object":
        dev_std_name_628fb,

    "test_prediction_object":
        test_pred_name_628fb,

    "test_mc_std_object":
        test_std_name_628fb,

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "abstention_rule_created":
        False,
}


# ============================================================
# 13. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Transferred uncertainty recipe : det_pred_pct_b5"
)

print(
    "Recipe re-selected on FD002     : False"
)

print(
    "Phase reference fitted on       : FD002 DEV only"
)

print(
    "Requested / actual phase bins   :",
    FD002_REQUESTED_PHASE_BINS_628FB,
    "/",
    FD002_ACTUAL_PHASE_BINS_628FB
)

print(
    "DEV uncertainty values          :",
    len(
        FD002_DEV_ADJUSTED_UNCERTAINTY_628FB
    )
)

print(
    "TEST uncertainty values         :",
    len(
        FD002_TEST_ADJUSTED_UNCERTAINTY_628FB
    )
)

print(
    "DEV adjusted uncertainty finite :",
    bool(
        np.isfinite(
            FD002_DEV_ADJUSTED_UNCERTAINTY_628FB
        ).all()
    )
)

print(
    "TEST adjusted uncertainty finite:",
    bool(
        np.isfinite(
            FD002_TEST_ADJUSTED_UNCERTAINTY_628FB
        ).all()
    )
)

print(
    "DEV values within [0,1]         :",
    dev_range_ok_628fb
)

print(
    "TEST values within [0,1]        :",
    test_range_ok_628fb
)

print(
    "Official FD002 TEST labels read : False"
)

print(
    "TEST scored                     : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Abstention rule created         : False"
)

print(
    "Model retrained                 : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.28F-B ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-B-R1
# FD002 EXACT SOURCE-ROLE REPAIR FOR PHASE-ADJUSTED UNCERTAINTY
#
# FD001 source-role replication:
#
#   FD001 DEVELOPMENT  -> FD002 FIT
#   FD001 VALIDATION   -> FD002 DEV
#
# Frozen recipe:
#   det_pred_pct_b5
#
# FIT:
#   deterministic prediction + MC-dropout STD
#   -> fit 5 quantile phase bins
#   -> fit within-phase uncertainty references
#
# DEV / TEST:
#   -> APPLY frozen FIT references only
#
# NO model retraining
# NO official FD002 TEST labels
# NO TEST scoring
# NO threshold selection
# NO files written
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B-R1 — EXACT SOURCE-ROLE REPAIR\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_628fbr1 = [
    "MODEL_FD002_628EB",
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "X_FIT_FD002_628D",
    "X_DEV_FD002_628D",
    "X_TEST_FD002_628D",
    "DEV_DETERMINISTIC_RECHECK_628FAR1",
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
    "DEV_MC_STD_FD002_628FAR2",
    "TEST_MC_STD_FD002_628FAR2",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628fbr1 in required_628fbr1:

    exists_628fbr1 = (
        name_628fbr1 in globals()
    )

    print(
        f"{name_628fbr1:<48}: "
        f"{exists_628fbr1}"
    )

    if not exists_628fbr1:

        raise RuntimeError(
            f"Missing required object: "
            f"{name_628fbr1}"
        )


# ============================================================
# 2. Frozen inference contract
# ============================================================

DET_BATCH_628FBR1 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "validation_batch_size"
    ]
)

MC_BATCH_628FBR1 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_batch_size"
    ]
)

MC_PASSES_628FBR1 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_passes"
    ]
)

MC_SEED_628FBR1 = int(
    FD002_GRU_TRAINING_CONTRACT_628EA[
        "mc_seed"
    ]
)


print("\n--- FROZEN INFERENCE CONTRACT ---")

print(
    "Deterministic batch :",
    DET_BATCH_628FBR1
)

print(
    "MC batch            :",
    MC_BATCH_628FBR1
)

print(
    "MC passes           :",
    MC_PASSES_628FBR1
)

print(
    "MC seed             :",
    MC_SEED_628FBR1
)

print(
    "MC STD ddof         : 0"
)


if DET_BATCH_628FBR1 != 256:
    raise RuntimeError(
        "Expected deterministic batch = 256."
    )

if MC_BATCH_628FBR1 != 512:
    raise RuntimeError(
        "Expected MC batch = 512."
    )

if MC_PASSES_628FBR1 != 50:
    raise RuntimeError(
        "Expected MC passes = 50."
    )

if MC_SEED_628FBR1 != 42:
    raise RuntimeError(
        "Expected MC seed = 42."
    )


# ============================================================
# 3. Arrays + device
# ============================================================

DEVICE_628FBR1 = next(
    MODEL_FD002_628EB.parameters()
).device


X_fit_628fbr1 = np.asarray(
    X_FIT_FD002_628D,
    dtype=np.float32
)

X_dev_628fbr1 = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32
)

X_test_628fbr1 = np.asarray(
    X_TEST_FD002_628D,
    dtype=np.float32
)


print("\n--- ARRAY CONTRACT ---")

print(
    "Device :",
    DEVICE_628FBR1
)

print(
    "FIT  :",
    X_fit_628fbr1.shape
)

print(
    "DEV  :",
    X_dev_628fbr1.shape
)

print(
    "TEST :",
    X_test_628fbr1.shape
)


if X_fit_628fbr1.shape != (
    37852,
    30,
    17
):
    raise RuntimeError(
        "Unexpected FIT array shape."
    )

if X_dev_628fbr1.shape != (
    8367,
    30,
    17
):
    raise RuntimeError(
        "Unexpected DEV array shape."
    )

if X_test_628fbr1.shape != (
    259,
    30,
    17
):
    raise RuntimeError(
        "Unexpected TEST array shape."
    )


# ============================================================
# 4. Helper
# ============================================================

def vec1d_628fbr1(x):

    if hasattr(
        x,
        "detach"
    ):
        x = (
            x
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


# ============================================================
# 5. Deterministic FIT prediction
#
# Exact frozen deterministic inference batch = 256.
# ============================================================

fit_det_dataset_628fbr1 = TensorDataset(
    torch.from_numpy(
        X_fit_628fbr1
    )
)

fit_det_loader_628fbr1 = DataLoader(
    fit_det_dataset_628fbr1,
    batch_size=DET_BATCH_628FBR1,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE_628FBR1.type
        ==
        "cuda"
    ),
)


@torch.inference_mode()
def deterministic_predict_628fbr1(
    model,
    loader
):

    model.eval()

    outputs_628fbr1 = []

    for (xb_628fbr1,) in loader:

        xb_628fbr1 = xb_628fbr1.to(
            DEVICE_628FBR1,
            non_blocking=True
        )

        pred_628fbr1 = model(
            xb_628fbr1
        )

        outputs_628fbr1.append(
            pred_628fbr1
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

    return (
        np.concatenate(
            outputs_628fbr1
        )
        .astype(
            np.float64
        )
    )


FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1 = (
    deterministic_predict_628fbr1(
        MODEL_FD002_628EB,
        fit_det_loader_628fbr1
    )
)


if (
    len(
        FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
    )
    !=
    37852
):

    raise RuntimeError(
        "FIT prediction count mismatch."
    )


if not np.isfinite(
    FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
).all():

    raise RuntimeError(
        "Non-finite FIT predictions."
    )


print(
    "\n--- FIT DETERMINISTIC PREDICTIONS ---"
)

print(
    "Count   :",
    len(
        FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
    )
)

print(
    "Minimum :",
    float(
        np.min(
            FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
        )
    )
)

print(
    "Median  :",
    float(
        np.median(
            FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
        )
    )
)

print(
    "Mean    :",
    float(
        np.mean(
            FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
        )
    )
)

print(
    "Maximum :",
    float(
        np.max(
            FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1
        )
    )
)


# ============================================================
# 6. FIT MC-dropout loader
# ============================================================

fit_mc_dataset_628fbr1 = TensorDataset(
    torch.from_numpy(
        X_fit_628fbr1
    )
)

fit_mc_loader_628fbr1 = DataLoader(
    fit_mc_dataset_628fbr1,
    batch_size=MC_BATCH_628FBR1,
    shuffle=False,
    num_workers=0,
    pin_memory=(
        DEVICE_628FBR1.type
        ==
        "cuda"
    ),
)


# ============================================================
# 7. MC-dropout helper
# ============================================================

@torch.inference_mode()
def mc_dropout_predict_628fbr1(
    model,
    loader,
    n_samples,
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    n_obs_628fbr1 = len(
        loader.dataset
    )

    mc_matrix_628fbr1 = np.empty(
        (
            n_samples,
            n_obs_628fbr1
        ),
        dtype=np.float32
    )

    model.train()

    for pass_idx_628fbr1 in range(
        n_samples
    ):

        preds_628fbr1 = []

        for (xb_628fbr1,) in loader:

            xb_628fbr1 = xb_628fbr1.to(
                DEVICE_628FBR1,
                non_blocking=True
            )

            pred_628fbr1 = model(
                xb_628fbr1
            )

            preds_628fbr1.append(
                pred_628fbr1
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
                .astype(
                    np.float32
                )
            )

        pass_predictions_628fbr1 = (
            np.concatenate(
                preds_628fbr1
            )
        )

        if (
            len(
                pass_predictions_628fbr1
            )
            !=
            n_obs_628fbr1
        ):

            raise RuntimeError(
                "FIT MC prediction count mismatch."
            )

        mc_matrix_628fbr1[
            pass_idx_628fbr1
        ] = pass_predictions_628fbr1

    model.eval()

    mc_mean_628fbr1 = (
        mc_matrix_628fbr1
        .mean(
            axis=0
        )
        .astype(
            np.float64
        )
    )

    mc_std_628fbr1 = (
        mc_matrix_628fbr1
        .std(
            axis=0,
            ddof=0
        )
        .astype(
            np.float64
        )
    )

    return (
        mc_matrix_628fbr1,
        mc_mean_628fbr1,
        mc_std_628fbr1
    )


# ============================================================
# 8. Compute FIT MC-dropout uncertainty
# ============================================================

print(
    "\n--- MC-DROPOUT: FD002 FIT ---"
)

print(
    f"Running {MC_PASSES_628FBR1} "
    "stochastic forward passes..."
)


(
    FIT_MC_MATRIX_FD002_628FBR1,
    FIT_MC_MEAN_FD002_628FBR1,
    FIT_MC_STD_FD002_628FBR1

) = mc_dropout_predict_628fbr1(

    MODEL_FD002_628EB,

    fit_mc_loader_628fbr1,

    n_samples=MC_PASSES_628FBR1,

    seed=MC_SEED_628FBR1
)


if not np.isfinite(
    FIT_MC_STD_FD002_628FBR1
).all():

    raise RuntimeError(
        "FIT MC-STD contains non-finite values."
    )


print(
    "\n--- FIT RAW MC-STD ---"
)

print(
    "Minimum :",
    float(
        np.min(
            FIT_MC_STD_FD002_628FBR1
        )
    )
)

print(
    "Median  :",
    float(
        np.median(
            FIT_MC_STD_FD002_628FBR1
        )
    )
)

print(
    "Mean    :",
    float(
        np.mean(
            FIT_MC_STD_FD002_628FBR1
        )
    )
)

print(
    "Maximum :",
    float(
        np.max(
            FIT_MC_STD_FD002_628FBR1
        )
    )
)


# ============================================================
# 9. Existing DEV + TEST clean inputs
# ============================================================

DEV_DET_628FBR1 = vec1d_628fbr1(
    DEV_DETERMINISTIC_RECHECK_628FAR1
)

TEST_DET_628FBR1 = vec1d_628fbr1(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2
)

DEV_MC_STD_628FBR1 = vec1d_628fbr1(
    DEV_MC_STD_FD002_628FAR2
)

TEST_MC_STD_628FBR1 = vec1d_628fbr1(
    TEST_MC_STD_FD002_628FAR2
)


if len(
    DEV_DET_628FBR1
) != 8367:

    raise RuntimeError(
        "DEV deterministic length mismatch."
    )

if len(
    TEST_DET_628FBR1
) != 259:

    raise RuntimeError(
        "TEST deterministic length mismatch."
    )

if len(
    DEV_MC_STD_628FBR1
) != 8367:

    raise RuntimeError(
        "DEV MC-STD length mismatch."
    )

if len(
    TEST_MC_STD_628FBR1
) != 259:

    raise RuntimeError(
        "TEST MC-STD length mismatch."
    )


# ============================================================
# 10. Exact FD001 transferred phase fitting rule
#
# FIT ONLY.
# ============================================================

REQUESTED_BINS_628FBR1 = 5

quantiles_628fbr1 = np.linspace(
    0.0,
    1.0,
    REQUESTED_BINS_628FBR1 + 1
)

raw_edges_628fbr1 = np.quantile(
    FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1,
    quantiles_628fbr1
)

FD002_PHASE_EDGES_628FBR1 = np.unique(
    raw_edges_628fbr1[
        1:-1
    ]
)


fit_phase_id_628fbr1 = np.digitize(
    FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1,
    FD002_PHASE_EDGES_628FBR1,
    right=True
)


FD002_PHASE_REFERENCES_628FBR1 = {}

for phase_id_628fbr1 in np.unique(
    fit_phase_id_628fbr1
):

    values_628fbr1 = (
        FIT_MC_STD_FD002_628FBR1[
            fit_phase_id_628fbr1
            ==
            phase_id_628fbr1
        ]
    )

    FD002_PHASE_REFERENCES_628FBR1[
        int(
            phase_id_628fbr1
        )
    ] = np.sort(
        values_628fbr1
    )


ACTUAL_BINS_628FBR1 = len(
    FD002_PHASE_REFERENCES_628FBR1
)


# ============================================================
# 11. Exact empirical mid-rank application
# ============================================================

def apply_phase_adjustment_628fbr1(
    prediction,
    mc_std
):

    prediction_628fbr1 = (
        vec1d_628fbr1(
            prediction
        )
    )

    uncertainty_628fbr1 = (
        vec1d_628fbr1(
            mc_std
        )
    )

    if (
        prediction_628fbr1.shape
        !=
        uncertainty_628fbr1.shape
    ):

        raise ValueError(
            "Prediction and uncertainty "
            "shape mismatch."
        )

    phase_id_628fbr1 = np.digitize(
        prediction_628fbr1,
        FD002_PHASE_EDGES_628FBR1,
        right=True
    )

    result_628fbr1 = np.zeros(
        len(
            prediction_628fbr1
        ),
        dtype=np.float64
    )

    for b_628fbr1 in np.unique(
        phase_id_628fbr1
    ):

        mask_628fbr1 = (
            phase_id_628fbr1
            ==
            b_628fbr1
        )

        if (
            int(
                b_628fbr1
            )
            not in
            FD002_PHASE_REFERENCES_628FBR1
        ):

            raise RuntimeError(
                "Missing FIT reference for "
                f"phase {b_628fbr1}."
            )

        reference_628fbr1 = (
            FD002_PHASE_REFERENCES_628FBR1[
                int(
                    b_628fbr1
                )
            ]
        )

        x_628fbr1 = (
            uncertainty_628fbr1[
                mask_628fbr1
            ]
        )

        left_628fbr1 = np.searchsorted(
            reference_628fbr1,
            x_628fbr1,
            side="left"
        )

        right_628fbr1 = np.searchsorted(
            reference_628fbr1,
            x_628fbr1,
            side="right"
        )

        result_628fbr1[
            mask_628fbr1
        ] = (
            left_628fbr1
            +
            right_628fbr1
        ) / (
            2.0
            *
            len(
                reference_628fbr1
            )
        )

    return result_628fbr1


# ============================================================
# 12. Correct phase-adjusted uncertainty
# ============================================================

FD002_FIT_ADJUSTED_UNCERTAINTY_628FBR1 = (
    apply_phase_adjustment_628fbr1(
        FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1,
        FIT_MC_STD_FD002_628FBR1
    )
)


FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1 = (
    apply_phase_adjustment_628fbr1(
        DEV_DET_628FBR1,
        DEV_MC_STD_628FBR1
    )
)


FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1 = (
    apply_phase_adjustment_628fbr1(
        TEST_DET_628FBR1,
        TEST_MC_STD_628FBR1
    )
)


# ============================================================
# 13. Reference contract
# ============================================================

print(
    "\n--- CORRECT FITTED PHASE REFERENCE ---"
)

print(
    "Reference source       : FD002 FIT only"
)

print(
    "Transferred recipe     : det_pred_pct_b5"
)

print(
    "Requested bins         :",
    REQUESTED_BINS_628FBR1
)

print(
    "Actual bins            :",
    ACTUAL_BINS_628FBR1
)

print(
    "Interior edges         :",
    FD002_PHASE_EDGES_628FBR1.tolist()
)


print(
    "\n--- FIT REFERENCE BIN SIZES ---"
)

for b_628fbr1 in sorted(
    FD002_PHASE_REFERENCES_628FBR1
):

    print(
        f"Phase {b_628fbr1}:",
        len(
            FD002_PHASE_REFERENCES_628FBR1[
                b_628fbr1
            ]
        )
    )


# ============================================================
# 14. Distribution audit
# ============================================================

def summary_628fbr1(
    population,
    values
):

    values_628fbr1 = vec1d_628fbr1(
        values
    )

    return {
        "population":
            population,

        "n":
            len(
                values_628fbr1
            ),

        "minimum":
            float(
                np.min(
                    values_628fbr1
                )
            ),

        "q25":
            float(
                np.quantile(
                    values_628fbr1,
                    0.25
                )
            ),

        "median":
            float(
                np.median(
                    values_628fbr1
                )
            ),

        "mean":
            float(
                np.mean(
                    values_628fbr1
                )
            ),

        "q75":
            float(
                np.quantile(
                    values_628fbr1,
                    0.75
                )
            ),

        "p95":
            float(
                np.quantile(
                    values_628fbr1,
                    0.95
                )
            ),

        "maximum":
            float(
                np.max(
                    values_628fbr1
                )
            ),

        "finite":
            bool(
                np.isfinite(
                    values_628fbr1
                ).all()
            ),
    }


FD002_PHASE_ADJUSTED_SUMMARY_628FBR1 = (
    pd.DataFrame(
        [
            summary_628fbr1(
                "FIT",
                FD002_FIT_ADJUSTED_UNCERTAINTY_628FBR1
            ),

            summary_628fbr1(
                "DEV",
                FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
            ),

            summary_628fbr1(
                "TEST",
                FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1
            ),
        ]
    )
)


print(
    "\n--- CORRECTED PHASE-ADJUSTED DISTRIBUTIONS ---"
)

print(
    FD002_PHASE_ADJUSTED_SUMMARY_628FBR1
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 15. Quantify difference from provisional DEV-fitted result
# ============================================================

if (
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FB"
    in globals()
):

    old_dev_628fbr1 = vec1d_628fbr1(
        FD002_DEV_ADJUSTED_UNCERTAINTY_628FB
    )

    dev_change_628fbr1 = np.abs(
        FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
        -
        old_dev_628fbr1
    )

    print(
        "\n--- PROVISIONAL-vs-CORRECTED DEV DIFFERENCE ---"
    )

    print(
        "Mean absolute difference :",
        float(
            np.mean(
                dev_change_628fbr1
            )
        )
    )

    print(
        "Maximum absolute difference:",
        float(
            np.max(
                dev_change_628fbr1
            )
        )
    )


if (
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB"
    in globals()
):

    old_test_628fbr1 = vec1d_628fbr1(
        FD002_TEST_ADJUSTED_UNCERTAINTY_628FB
    )

    test_change_628fbr1 = np.abs(
        FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1
        -
        old_test_628fbr1
    )

    print(
        "\n--- PROVISIONAL-vs-CORRECTED TEST DIFFERENCE ---"
    )

    print(
        "Mean absolute difference :",
        float(
            np.mean(
                test_change_628fbr1
            )
        )
    )

    print(
        "Maximum absolute difference:",
        float(
            np.max(
                test_change_628fbr1
            )
        )
    )


# ============================================================
# 16. Integrity
# ============================================================

for label_628fbr1, arr_628fbr1 in [

    (
        "FIT",
        FD002_FIT_ADJUSTED_UNCERTAINTY_628FBR1
    ),

    (
        "DEV",
        FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
    ),

    (
        "TEST",
        FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1
    ),

]:

    if not np.isfinite(
        arr_628fbr1
    ).all():

        raise RuntimeError(
            f"{label_628fbr1} adjusted "
            "uncertainty is non-finite."
        )

    if not np.all(
        (
            arr_628fbr1
            >=
            0.0
        )
        &
        (
            arr_628fbr1
            <=
            1.0
        )
    ):

        raise RuntimeError(
            f"{label_628fbr1} adjusted "
            "uncertainty outside [0,1]."
        )


# ============================================================
# 17. Correct frozen in-memory contract
# ============================================================

FD002_UNCERTAINTY_RECIPE_628FBR1 = {

    "dataset":
        "FD002",

    "candidate":
        "det_pred_pct_b5",

    "recipe_origin":
        "transferred_from_FD001",

    "role_mapping":
        {
            "FD001_DEVELOPMENT":
                "FD002_FIT",

            "FD001_VALIDATION":
                "FD002_DEV",
        },

    "phase_reference_fit_source":
        "FD002_FIT_only",

    "phase_source":
        "deterministic_prediction",

    "requested_bins":
        5,

    "actual_bins":
        ACTUAL_BINS_628FBR1,

    "phase_edges":
        FD002_PHASE_EDGES_628FBR1.copy(),

    "base_uncertainty":
        "MC_dropout_STD",

    "mc_passes":
        MC_PASSES_628FBR1,

    "mc_std_ddof":
        0,

    "phase_assignment":
        "np.digitize_right=True",

    "within_phase_mapping":
        "empirical_midrank_percentile",

    "provisional_628FB_superseded":
        True,

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 18. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-B-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 DEVELOPMENT role mapped to : FD002 FIT"
)

print(
    "FD001 VALIDATION role mapped to  : FD002 DEV"
)

print(
    "FIT reference windows            :",
    len(
        FD002_FIT_ADJUSTED_UNCERTAINTY_628FBR1
    )
)

print(
    "DEV transformed windows          :",
    len(
        FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
    )
)

print(
    "TEST transformed endpoints       :",
    len(
        FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1
    )
)

print(
    "Requested / actual phase bins    :",
    REQUESTED_BINS_628FBR1,
    "/",
    ACTUAL_BINS_628FBR1
)

print(
    "Phase reference fitted on        : FD002 FIT only"
)

print(
    "Recipe re-selected on FD002      : False"
)

print(
    "Provisional Step 6.28F-B superseded:",
    True
)

print(
    "Official FD002 TEST labels read  : False"
)

print(
    "TEST scored                      : False"
)

print(
    "Threshold selected               : False"
)

print(
    "Model retrained                  : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.28F-B-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-C
# FD002 TWO-STAGE RELIABILITY-RISK CALIBRATION
#
# Frozen methodology transferred from FD001:
#
#   Phase-adjusted uncertainty
#       -> G1 right empirical CDF using all clean DEV windows
#       -> DEV endpoint extraction
#       -> G2 empirical mid-rank CDF using clean DEV endpoints
#
# G1 reference:
#       all 8,367 clean FD002 DEV windows
#
# G2 reference:
#       one clean endpoint from each of 52 FD002 DEV engines
#
# TEST:
#       transformation only
#
# IMPORTANT:
# - No official FD002 TEST labels
# - No TEST scoring
# - No corruption applied yet
# - No gate comparison
# - No threshold selection
# - No abstention
# - No model retraining
# - No files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C — FD002 G1/G2 RELIABILITY-RISK CALIBRATION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_628fc = [
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    "DEV_WINDOW_KEYS_FD002_628D",
    "TEST_ENDPOINT_META_FD002_628D",
    "FD002_UNCERTAINTY_RECIPE_628FBR1",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628fc in required_628fc:

    exists_628fc = (
        name_628fc in globals()
    )

    print(
        f"{name_628fc:<48}: "
        f"{exists_628fc}"
    )

    if not exists_628fc:

        raise RuntimeError(
            "Missing required object: "
            + name_628fc
        )


# ============================================================
# 2. Helper
# ============================================================

def vec1d_628fc(x):

    if hasattr(
        x,
        "detach"
    ):
        x = (
            x.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


dev_adjusted_628fc = vec1d_628fc(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
)

test_adjusted_628fc = vec1d_628fc(
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1
)


if len(dev_adjusted_628fc) != 8367:

    raise RuntimeError(
        "Unexpected DEV adjusted-uncertainty length."
    )


if len(test_adjusted_628fc) != 259:

    raise RuntimeError(
        "Unexpected TEST adjusted-uncertainty length."
    )


if not np.isfinite(
    dev_adjusted_628fc
).all():

    raise RuntimeError(
        "DEV adjusted uncertainty contains non-finite values."
    )


if not np.isfinite(
    test_adjusted_628fc
).all():

    raise RuntimeError(
        "TEST adjusted uncertainty contains non-finite values."
    )


# ============================================================
# 3. DEV window-key contract
# ============================================================

dev_keys_628fc = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)


print(
    "\n--- DEV WINDOW-KEY CONTRACT ---"
)

print(
    "Shape   :",
    dev_keys_628fc.shape
)

print(
    "Columns :",
    dev_keys_628fc.columns.tolist()
)


required_key_columns_628fc = [
    "unit_id",
    "cycle",
]


for col_628fc in required_key_columns_628fc:

    if col_628fc not in dev_keys_628fc.columns:

        raise RuntimeError(
            "Missing DEV window-key column: "
            + col_628fc
        )


if len(dev_keys_628fc) != len(
    dev_adjusted_628fc
):

    raise RuntimeError(
        "DEV window keys are not aligned with "
        "DEV adjusted uncertainty."
    )


duplicate_keys_628fc = int(
    dev_keys_628fc
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


print(
    "Duplicate unit/cycle keys :",
    duplicate_keys_628fc
)


if duplicate_keys_628fc != 0:

    raise RuntimeError(
        "Duplicate DEV endpoint keys detected."
    )


# ============================================================
# 4. G1 reference
#
# Exact transferred rule:
# right empirical CDF
#
# G1(x) = count(reference <= x) / N
#
# Reference source = clean FD002 DEV windows only.
# ============================================================

G1_REFERENCE_FD002_628FC = np.sort(
    dev_adjusted_628fc.copy()
)


def apply_g1_right_ecdf_628fc(
    values,
    reference=G1_REFERENCE_FD002_628FC,
):

    values_628fc = vec1d_628fc(
        values
    )

    reference_628fc = vec1d_628fc(
        reference
    )

    if len(reference_628fc) == 0:

        raise ValueError(
            "G1 reference cannot be empty."
        )

    if not np.isfinite(
        reference_628fc
    ).all():

        raise ValueError(
            "G1 reference contains non-finite values."
        )

    if not np.isfinite(
        values_628fc
    ).all():

        raise ValueError(
            "G1 input contains non-finite values."
        )

    return (
        np.searchsorted(
            reference_628fc,
            values_628fc,
            side="right"
        )
        /
        float(
            len(reference_628fc)
        )
    )


FD002_DEV_G1_UNCERTAINTY_RISK_628FC = (
    apply_g1_right_ecdf_628fc(
        dev_adjusted_628fc
    )
)


FD002_TEST_G1_UNCERTAINTY_RISK_628FC = (
    apply_g1_right_ecdf_628fc(
        test_adjusted_628fc
    )
)


# ============================================================
# 5. Identify exactly one final DEV endpoint per engine
# ============================================================

dev_keys_with_position_628fc = (
    dev_keys_628fc.copy()
)

dev_keys_with_position_628fc[
    "_row_position"
] = np.arange(
    len(
        dev_keys_with_position_628fc
    ),
    dtype=int
)


dev_endpoint_rows_628fc = (
    dev_keys_with_position_628fc
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .groupby(
        "unit_id",
        as_index=False
    )
    .tail(1)
    .sort_values(
        "unit_id"
    )
    .reset_index(drop=True)
)


DEV_ENDPOINT_POSITIONS_FD002_628FC = (
    dev_endpoint_rows_628fc[
        "_row_position"
    ]
    .to_numpy(
        dtype=int
    )
)


print(
    "\n--- DEV ENDPOINT CONTRACT ---"
)

print(
    "DEV endpoint rows       :",
    len(
        dev_endpoint_rows_628fc
    )
)

print(
    "Unique DEV engines      :",
    dev_endpoint_rows_628fc[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate DEV engines   :",
    int(
        dev_endpoint_rows_628fc[
            "unit_id"
        ]
        .duplicated()
        .sum()
    )
)


if len(
    dev_endpoint_rows_628fc
) != 52:

    raise RuntimeError(
        "Expected exactly 52 DEV endpoints."
    )


if (
    dev_endpoint_rows_628fc[
        "unit_id"
    ].nunique()
    !=
    52
):

    raise RuntimeError(
        "Expected exactly 52 unique DEV engines."
    )


# ============================================================
# 6. DEV endpoint stage-1 risk
# ============================================================

FD002_DEV_ENDPOINT_G1_RISK_628FC = (
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC[
        DEV_ENDPOINT_POSITIONS_FD002_628FC
    ]
)


DEV_ENDPOINT_TABLE_FD002_628FC = (
    dev_endpoint_rows_628fc[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .copy()
)


DEV_ENDPOINT_TABLE_FD002_628FC[
    "adjusted_uncertainty"
] = (
    dev_adjusted_628fc[
        DEV_ENDPOINT_POSITIONS_FD002_628FC
    ]
)


DEV_ENDPOINT_TABLE_FD002_628FC[
    "g1_uncertainty_risk"
] = (
    FD002_DEV_ENDPOINT_G1_RISK_628FC
)


# ============================================================
# 7. G2 endpoint reference
#
# Reference = clean G1 risk of the 52 DEV endpoints.
#
# Exact transferred G2 rule:
#
# mid-rank empirical CDF
#
# G2(x) =
#   [count(reference < x)
#    + count(reference <= x)] / (2*N)
# ============================================================

G2_REFERENCE_FD002_628FC = np.sort(
    FD002_DEV_ENDPOINT_G1_RISK_628FC.copy()
)


def apply_g2_midrank_628fc(
    values,
    reference=G2_REFERENCE_FD002_628FC,
):

    values_628fc = vec1d_628fc(
        values
    )

    reference_628fc = vec1d_628fc(
        reference
    )

    if len(reference_628fc) == 0:

        raise ValueError(
            "G2 reference cannot be empty."
        )

    if not np.isfinite(
        reference_628fc
    ).all():

        raise ValueError(
            "G2 reference contains non-finite values."
        )

    if not np.isfinite(
        values_628fc
    ).all():

        raise ValueError(
            "G2 input contains non-finite values."
        )

    left_628fc = np.searchsorted(
        reference_628fc,
        values_628fc,
        side="left"
    )

    right_628fc = np.searchsorted(
        reference_628fc,
        values_628fc,
        side="right"
    )

    return (
        left_628fc
        +
        right_628fc
    ) / (
        2.0
        *
        len(reference_628fc)
    )


FD002_DEV_ENDPOINT_SELECTIVE_RISK_628FC = (
    apply_g2_midrank_628fc(
        FD002_DEV_ENDPOINT_G1_RISK_628FC
    )
)


FD002_TEST_SELECTIVE_RISK_628FC = (
    apply_g2_midrank_628fc(
        FD002_TEST_G1_UNCERTAINTY_RISK_628FC
    )
)


DEV_ENDPOINT_TABLE_FD002_628FC[
    "selective_risk"
] = (
    FD002_DEV_ENDPOINT_SELECTIVE_RISK_628FC
)


# ============================================================
# 8. TEST structural contract
# ============================================================

test_meta_628fc = (
    TEST_ENDPOINT_META_FD002_628D
    .copy()
    .reset_index(drop=True)
)


print(
    "\n--- TEST ENDPOINT CONTRACT ---"
)

print(
    "TEST metadata rows :",
    len(
        test_meta_628fc
    )
)

print(
    "TEST uncertainty   :",
    len(
        test_adjusted_628fc
    )
)

print(
    "TEST G1 risk       :",
    len(
        FD002_TEST_G1_UNCERTAINTY_RISK_628FC
    )
)

print(
    "TEST final risk    :",
    len(
        FD002_TEST_SELECTIVE_RISK_628FC
    )
)


if len(
    test_meta_628fc
) != 259:

    raise RuntimeError(
        "Expected 259 TEST endpoint metadata rows."
    )


# ============================================================
# 9. Build label-free TEST reliability table
# ============================================================

TEST_RELIABILITY_TABLE_FD002_628FC = (
    test_meta_628fc.copy()
)


TEST_RELIABILITY_TABLE_FD002_628FC[
    "adjusted_uncertainty"
] = (
    test_adjusted_628fc
)


TEST_RELIABILITY_TABLE_FD002_628FC[
    "g1_uncertainty_risk"
] = (
    FD002_TEST_G1_UNCERTAINTY_RISK_628FC
)


TEST_RELIABILITY_TABLE_FD002_628FC[
    "selective_risk"
] = (
    FD002_TEST_SELECTIVE_RISK_628FC
)


# ============================================================
# 10. Numerical integrity
# ============================================================

risk_objects_628fc = {
    "DEV_G1":
        FD002_DEV_G1_UNCERTAINTY_RISK_628FC,

    "DEV_ENDPOINT_G1":
        FD002_DEV_ENDPOINT_G1_RISK_628FC,

    "DEV_ENDPOINT_G2":
        FD002_DEV_ENDPOINT_SELECTIVE_RISK_628FC,

    "TEST_G1":
        FD002_TEST_G1_UNCERTAINTY_RISK_628FC,

    "TEST_G2":
        FD002_TEST_SELECTIVE_RISK_628FC,
}


print(
    "\n--- NUMERICAL INTEGRITY ---"
)


for name_628fc, values_628fc in (
    risk_objects_628fc.items()
):

    values_628fc = vec1d_628fc(
        values_628fc
    )

    finite_628fc = bool(
        np.isfinite(
            values_628fc
        ).all()
    )

    range_ok_628fc = bool(
        np.all(
            (
                values_628fc
                >=
                0.0
            )
            &
            (
                values_628fc
                <=
                1.0
            )
        )
    )

    print(
        f"{name_628fc:<20} "
        f"n={len(values_628fc):<6} "
        f"finite={finite_628fc} "
        f"range_ok={range_ok_628fc} "
        f"min={np.min(values_628fc):.6f} "
        f"median={np.median(values_628fc):.6f} "
        f"mean={np.mean(values_628fc):.6f} "
        f"max={np.max(values_628fc):.6f}"
    )

    if not finite_628fc:

        raise RuntimeError(
            name_628fc
            +
            " contains non-finite values."
        )

    if not range_ok_628fc:

        raise RuntimeError(
            name_628fc
            +
            " is outside [0,1]."
        )


# ============================================================
# 11. Reference summaries
# ============================================================

print(
    "\n--- G1 REFERENCE ---"
)

print(
    "Source : all clean FD002 DEV windows"
)

print(
    "Length :",
    len(
        G1_REFERENCE_FD002_628FC
    )
)

print(
    "Min    :",
    float(
        np.min(
            G1_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Median :",
    float(
        np.median(
            G1_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Mean   :",
    float(
        np.mean(
            G1_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Max    :",
    float(
        np.max(
            G1_REFERENCE_FD002_628FC
        )
    )
)


print(
    "\n--- G2 REFERENCE ---"
)

print(
    "Source : 52 clean FD002 DEV endpoints"
)

print(
    "Length :",
    len(
        G2_REFERENCE_FD002_628FC
    )
)

print(
    "Min    :",
    float(
        np.min(
            G2_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Median :",
    float(
        np.median(
            G2_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Mean   :",
    float(
        np.mean(
            G2_REFERENCE_FD002_628FC
        )
    )
)

print(
    "Max    :",
    float(
        np.max(
            G2_REFERENCE_FD002_628FC
        )
    )
)


# ============================================================
# 12. Saturation diagnostic
#
# This is inspection only.
# It does NOT imply an operating threshold.
# ============================================================

test_g1_628fc = (
    FD002_TEST_G1_UNCERTAINTY_RISK_628FC
)

g2_max_628fc = float(
    np.max(
        G2_REFERENCE_FD002_628FC
    )
)


above_g2_max_628fc = int(
    np.sum(
        test_g1_628fc
        >
        g2_max_628fc
    )
)


equal_g2_max_628fc = int(
    np.sum(
        np.isclose(
            test_g1_628fc,
            g2_max_628fc,
            rtol=0.0,
            atol=1e-15
        )
    )
)


final_risk_one_628fc = int(
    np.sum(
        np.isclose(
            FD002_TEST_SELECTIVE_RISK_628FC,
            1.0,
            rtol=0.0,
            atol=1e-15
        )
    )
)


print(
    "\n--- TEST G2 SATURATION DIAGNOSTIC ---"
)

print(
    "G2 reference maximum       :",
    g2_max_628fc
)

print(
    "TEST G1 > G2 ref maximum   :",
    above_g2_max_628fc
)

print(
    "TEST G1 = G2 ref maximum   :",
    equal_g2_max_628fc
)

print(
    "TEST final risk == 1.0     :",
    final_risk_one_628fc
)


# ============================================================
# 13. Freeze G1/G2 calibration contract
# ============================================================

FD002_G1_G2_CONTRACT_628FC = {

    "dataset":
        "FD002",

    "input_uncertainty":
        "phase_adjusted_uncertainty_628FBR1",

    "g1_reference_source":
        "all_clean_FD002_DEV_windows",

    "g1_reference_length":
        int(
            len(
                G1_REFERENCE_FD002_628FC
            )
        ),

    "g1_transform":
        "right_empirical_cdf",

    "g1_formula":
        "count(reference<=x)/N",

    "g2_reference_source":
        "clean_FD002_DEV_engine_endpoints",

    "g2_reference_length":
        int(
            len(
                G2_REFERENCE_FD002_628FC
            )
        ),

    "g2_transform":
        "empirical_midrank_cdf",

    "g2_formula":
        (
            "(count(reference<x)"
            "+count(reference<=x))/(2*N)"
        ),

    "dev_endpoint_count":
        52,

    "test_endpoint_count":
        259,

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "corruption_applied":
        False,

    "threshold_selected":
        False,

    "abstention_rule_created":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 14. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C FINAL SUMMARY\n"
    "============================================================"
)

print(
    "G1 reference source       : FD002 DEV clean windows"
)

print(
    "G1 reference length       :",
    len(
        G1_REFERENCE_FD002_628FC
    )
)

print(
    "G1 transformation         : right ECDF"
)

print(
    "G2 reference source       : FD002 DEV clean endpoints"
)

print(
    "G2 reference length       :",
    len(
        G2_REFERENCE_FD002_628FC
    )
)

print(
    "G2 transformation         : empirical mid-rank ECDF"
)

print(
    "DEV endpoint engines      :",
    len(
        FD002_DEV_ENDPOINT_G1_RISK_628FC
    )
)

print(
    "TEST endpoint engines     :",
    len(
        FD002_TEST_SELECTIVE_RISK_628FC
    )
)

print(
    "Official FD002 TEST labels read : False"
)

print(
    "TEST scored                     : False"
)

print(
    "Corruption applied              : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Abstention rule created         : False"
)

print(
    "Model retrained                 : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.28F-C ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-C-R1
# FD002 DEV-ONLY LIFECYCLE-DEPENDENCE DIAGNOSTIC
#
# PURPOSE
# -------
# Determine whether the current G2 endpoint calibration is
# inadvertently encoding lifecycle position rather than
# reliability.
#
# IMPORTANT:
# - FD002 DEV ONLY
# - NO official TEST labels
# - NO TEST scoring
# - NO corruption
# - NO threshold selection
# - NO model retraining
# - NO files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R1 — DEV-ONLY LIFECYCLE DEPENDENCE AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_628fcr1 = [
    "DEV_WINDOW_KEYS_FD002_628D",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_DETERMINISTIC_RECHECK_628FAR1",
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "DEV_ENDPOINT_POSITIONS_FD002_628FC",
    "G2_REFERENCE_FD002_628FC",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628fcr1 in required_628fcr1:

    exists_628fcr1 = (
        name_628fcr1 in globals()
    )

    print(
        f"{name_628fcr1:<48}: "
        f"{exists_628fcr1}"
    )

    if not exists_628fcr1:

        raise RuntimeError(
            "Missing required object: "
            + name_628fcr1
        )


# ============================================================
# 2. Helper
# ============================================================

def vec1d_628fcr1(x):

    if hasattr(x, "detach"):

        x = (
            x.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


# ============================================================
# 3. Recover aligned DEV vectors
# ============================================================

keys_628fcr1 = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)


target_628fcr1 = vec1d_628fcr1(
    Y_DEV_PRIMARY_FD002_628D
)


prediction_628fcr1 = vec1d_628fcr1(
    DEV_DETERMINISTIC_RECHECK_628FAR1
)


adjusted_u_628fcr1 = vec1d_628fcr1(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1
)


g1_risk_628fcr1 = vec1d_628fcr1(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC
)


n_dev_628fcr1 = len(
    keys_628fcr1
)


lengths_628fcr1 = {
    "keys":
        len(keys_628fcr1),

    "target":
        len(target_628fcr1),

    "prediction":
        len(prediction_628fcr1),

    "adjusted_uncertainty":
        len(adjusted_u_628fcr1),

    "g1_risk":
        len(g1_risk_628fcr1),
}


print(
    "\n--- ALIGNED DEV LENGTHS ---"
)

for k_628fcr1, v_628fcr1 in (
    lengths_628fcr1.items()
):

    print(
        f"{k_628fcr1:<24}: "
        f"{v_628fcr1}"
    )


if len(
    set(
        lengths_628fcr1.values()
    )
) != 1:

    raise RuntimeError(
        "DEV vectors are not aligned."
    )


# ============================================================
# 4. Build DEV diagnostic table
# ============================================================

DEV_LIFECYCLE_AUDIT_FD002_628FCR1 = (
    keys_628fcr1.copy()
)


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "target_rul"
] = target_628fcr1


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "prediction"
] = prediction_628fcr1


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "absolute_error"
] = np.abs(
    prediction_628fcr1
    -
    target_628fcr1
)


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "adjusted_uncertainty"
] = adjusted_u_628fcr1


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "g1_risk"
] = g1_risk_628fcr1


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "is_true_endpoint"
] = False


endpoint_positions_628fcr1 = np.asarray(
    DEV_ENDPOINT_POSITIONS_FD002_628FC,
    dtype=int
)


DEV_LIFECYCLE_AUDIT_FD002_628FCR1.loc[
    endpoint_positions_628fcr1,
    "is_true_endpoint"
] = True


# ============================================================
# 5. Apply CURRENT G2 reference to every DEV window
#
# This is diagnostic only.
# No operating threshold is being selected.
# ============================================================

g2_reference_628fcr1 = np.sort(
    vec1d_628fcr1(
        G2_REFERENCE_FD002_628FC
    )
)


def apply_current_g2_628fcr1(
    values
):

    values_628fcr1 = vec1d_628fcr1(
        values
    )

    left_628fcr1 = np.searchsorted(
        g2_reference_628fcr1,
        values_628fcr1,
        side="left"
    )

    right_628fcr1 = np.searchsorted(
        g2_reference_628fcr1,
        values_628fcr1,
        side="right"
    )

    return (
        left_628fcr1
        +
        right_628fcr1
    ) / (
        2.0
        *
        len(
            g2_reference_628fcr1
        )
    )


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "current_g2_risk"
] = apply_current_g2_628fcr1(
    g1_risk_628fcr1
)


# ============================================================
# 6. Endpoint contract audit
# ============================================================

endpoint_table_628fcr1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
            "is_true_endpoint"
        ]
    ]
    .copy()
)


print(
    "\n--- TRUE DEV ENDPOINT AUDIT ---"
)

print(
    "Endpoint rows             :",
    len(endpoint_table_628fcr1)
)

print(
    "Unique endpoint engines   :",
    endpoint_table_628fcr1[
        "unit_id"
    ].nunique()
)

print(
    "Endpoint target min       :",
    float(
        endpoint_table_628fcr1[
            "target_rul"
        ].min()
    )
)

print(
    "Endpoint target median    :",
    float(
        endpoint_table_628fcr1[
            "target_rul"
        ].median()
    )
)

print(
    "Endpoint target max       :",
    float(
        endpoint_table_628fcr1[
            "target_rul"
        ].max()
    )
)

print(
    "All endpoint targets = 0  :",
    bool(
        np.allclose(
            endpoint_table_628fcr1[
                "target_rul"
            ].to_numpy(),
            0.0,
            rtol=0.0,
            atol=0.0
        )
    )
)


# ============================================================
# 7. RUL-band analysis
# ============================================================

rul_edges_628fcr1 = [
    -0.5,
    5.0,
    10.0,
    20.0,
    40.0,
    60.0,
    80.0,
    100.0,
    124.999999,
    125.000001,
]


rul_labels_628fcr1 = [
    "0-5",
    "6-10",
    "11-20",
    "21-40",
    "41-60",
    "61-80",
    "81-100",
    "101-124",
    "125_cap",
]


DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "rul_band"
] = pd.cut(
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        "target_rul"
    ],
    bins=rul_edges_628fcr1,
    labels=rul_labels_628fcr1,
    include_lowest=True,
)


RUL_BAND_SUMMARY_FD002_628FCR1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    .groupby(
        "rul_band",
        observed=True
    )
    .agg(
        rows=(
            "unit_id",
            "size"
        ),

        engines=(
            "unit_id",
            "nunique"
        ),

        target_median=(
            "target_rul",
            "median"
        ),

        prediction_median=(
            "prediction",
            "median"
        ),

        adjusted_u_median=(
            "adjusted_uncertainty",
            "median"
        ),

        g1_median=(
            "g1_risk",
            "median"
        ),

        g1_mean=(
            "g1_risk",
            "mean"
        ),

        final_g2_median=(
            "current_g2_risk",
            "median"
        ),

        final_g2_mean=(
            "current_g2_risk",
            "mean"
        ),

        mae=(
            "absolute_error",
            "mean"
        ),
    )
    .reset_index()
)


# Saturation fraction within each RUL band.
saturation_628fcr1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    .assign(
        g2_saturated=lambda df:
        np.isclose(
            df[
                "current_g2_risk"
            ],
            1.0,
            rtol=0.0,
            atol=1e-15
        )
    )
    .groupby(
        "rul_band",
        observed=True
    )[
        "g2_saturated"
    ]
    .mean()
    .rename(
        "g2_saturation_fraction"
    )
    .reset_index()
)


RUL_BAND_SUMMARY_FD002_628FCR1 = (
    RUL_BAND_SUMMARY_FD002_628FCR1
    .merge(
        saturation_628fcr1,
        on="rul_band",
        how="left"
    )
)


print(
    "\n--- DEV RUL-BAND RELIABILITY AUDIT ---"
)

print(
    RUL_BAND_SUMMARY_FD002_628FCR1
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Prediction-quantile analysis
#
# This checks whether residual phase dependence remains even
# after the 5-bin phase adjustment.
# ============================================================

DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
    "prediction_quintile"
] = pd.qcut(
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        "prediction"
    ],
    q=5,
    labels=[
        "Q1_low_pred",
        "Q2",
        "Q3",
        "Q4",
        "Q5_high_pred",
    ],
    duplicates="drop"
)


PREDICTION_PHASE_SUMMARY_FD002_628FCR1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    .groupby(
        "prediction_quintile",
        observed=True
    )
    .agg(
        rows=(
            "unit_id",
            "size"
        ),

        prediction_min=(
            "prediction",
            "min"
        ),

        prediction_median=(
            "prediction",
            "median"
        ),

        prediction_max=(
            "prediction",
            "max"
        ),

        adjusted_u_median=(
            "adjusted_uncertainty",
            "median"
        ),

        g1_median=(
            "g1_risk",
            "median"
        ),

        g1_mean=(
            "g1_risk",
            "mean"
        ),

        final_g2_median=(
            "current_g2_risk",
            "median"
        ),
    )
    .reset_index()
)


print(
    "\n--- DEV PREDICTION-QUINTILE AUDIT ---"
)

print(
    PREDICTION_PHASE_SUMMARY_FD002_628FCR1
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 9. Rank-correlation audit
# ============================================================

corr_columns_628fcr1 = [
    "cycle",
    "target_rul",
    "prediction",
    "absolute_error",
    "adjusted_uncertainty",
    "g1_risk",
    "current_g2_risk",
]


SPEARMAN_LIFECYCLE_FD002_628FCR1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        corr_columns_628fcr1
    ]
    .corr(
        method="spearman"
    )
)


print(
    "\n--- DEV SPEARMAN CORRELATIONS ---"
)

print(
    SPEARMAN_LIFECYCLE_FD002_628FCR1
    .round(4)
    .to_string()
)


# ============================================================
# 10. Endpoint vs non-endpoint comparison
# ============================================================

ENDPOINT_COMPARISON_FD002_628FCR1 = (
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    .groupby(
        "is_true_endpoint"
    )
    .agg(
        rows=(
            "unit_id",
            "size"
        ),

        engines=(
            "unit_id",
            "nunique"
        ),

        target_median=(
            "target_rul",
            "median"
        ),

        prediction_median=(
            "prediction",
            "median"
        ),

        adjusted_u_median=(
            "adjusted_uncertainty",
            "median"
        ),

        g1_median=(
            "g1_risk",
            "median"
        ),

        g1_mean=(
            "g1_risk",
            "mean"
        ),

        current_g2_median=(
            "current_g2_risk",
            "median"
        ),

        mae=(
            "absolute_error",
            "mean"
        ),
    )
    .reset_index()
)


print(
    "\n--- TRUE ENDPOINT vs OTHER DEV WINDOWS ---"
)

print(
    ENDPOINT_COMPARISON_FD002_628FCR1
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Current G2 saturation within DEV itself
# ============================================================

dev_g2_saturated_628fcr1 = np.isclose(
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        "current_g2_risk"
    ].to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-15
)


print(
    "\n--- CURRENT G2 DEV-ONLY SATURATION ---"
)

print(
    "G2 endpoint-reference maximum :",
    float(
        np.max(
            g2_reference_628fcr1
        )
    )
)

print(
    "All DEV windows               :",
    len(
        DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    )
)

print(
    "DEV windows with final risk=1 :",
    int(
        np.sum(
            dev_g2_saturated_628fcr1
        )
    )
)

print(
    "True DEV endpoints risk=1     :",
    int(
        np.sum(
            np.isclose(
                endpoint_table_628fcr1[
                    "current_g2_risk"
                ].to_numpy(),
                1.0,
                rtol=0.0,
                atol=1e-15
            )
        )
    )
)


# ============================================================
# 12. Freeze diagnostic evidence only
# ============================================================

FD002_G2_LIFECYCLE_DIAGNOSTIC_628FCR1 = {

    "dataset":
        "FD002",

    "population":
        "DEV_only",

    "dev_rows":
        int(
            len(
                DEV_LIFECYCLE_AUDIT_FD002_628FCR1
            )
        ),

    "dev_engines":
        int(
            DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
                "unit_id"
            ].nunique()
        ),

    "true_endpoint_rows":
        int(
            len(
                endpoint_table_628fcr1
            )
        ),

    "all_endpoint_targets_zero":
        bool(
            np.allclose(
                endpoint_table_628fcr1[
                    "target_rul"
                ].to_numpy(),
                0.0,
                rtol=0.0,
                atol=0.0
            )
        ),

    "current_g2_reference_max":
        float(
            np.max(
                g2_reference_628fcr1
            )
        ),

    "dev_g2_saturated_rows":
        int(
            np.sum(
                dev_g2_saturated_628fcr1
            )
        ),

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "methodology_changed":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 13. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Diagnostic population           : FD002 DEV only"
)

print(
    "DEV rows                        :",
    len(
        DEV_LIFECYCLE_AUDIT_FD002_628FCR1
    )
)

print(
    "DEV engines                     :",
    DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
        "unit_id"
    ].nunique()
)

print(
    "True endpoint rows              :",
    len(
        endpoint_table_628fcr1
    )
)

print(
    "All true endpoint targets = 0   :",
    bool(
        np.allclose(
            endpoint_table_628fcr1[
                "target_rul"
            ].to_numpy(),
            0.0,
            rtol=0.0,
            atol=0.0
        )
    )
)

print(
    "Current G2 method changed       : False"
)

print(
    "Official FD002 TEST labels read : False"
)

print(
    "TEST scored                     : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Model retrained                 : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.28F-C-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-C-R2
# FD002 ENGINE-BALANCED PSEUDO-ENDPOINT G2 REPAIR
#
# PURPOSE
# -------
# Replace the degenerate true-failure-endpoint G2 reference.
#
# New calibration principle:
#
#   Every eligible clean DEV window is a possible
#   truncated inference endpoint.
#
#   Each DEV engine contributes EQUAL TOTAL WEIGHT.
#
# Therefore:
#
#   G1 risk
#       -> engine-balanced weighted mid-rank ECDF
#       -> corrected final selective risk
#
# IMPORTANT:
# - DEV-only calibration
# - TEST transformation is label-free only
# - NO official TEST labels
# - NO TEST scoring
# - NO corruption
# - NO threshold selection
# - NO model retraining
# - NO files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R2 — ENGINE-BALANCED PSEUDO-ENDPOINT G2\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_628fcr2 = [
    "DEV_WINDOW_KEYS_FD002_628D",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",
    "TEST_ENDPOINT_META_FD002_628D",
    "DEV_ENDPOINT_POSITIONS_FD002_628FC",
]


print("\n--- REQUIRED OBJECTS ---")

for name_628fcr2 in required_628fcr2:

    exists_628fcr2 = (
        name_628fcr2 in globals()
    )

    print(
        f"{name_628fcr2:<48}: "
        f"{exists_628fcr2}"
    )

    if not exists_628fcr2:

        raise RuntimeError(
            "Missing required object: "
            + name_628fcr2
        )


# ============================================================
# 2. Helper
# ============================================================

def vec1d_628fcr2(x):

    if hasattr(
        x,
        "detach"
    ):
        x = (
            x.detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


# ============================================================
# 3. Aligned clean DEV population
# ============================================================

dev_keys_628fcr2 = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)


dev_g1_628fcr2 = vec1d_628fcr2(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC
)


test_g1_628fcr2 = vec1d_628fcr2(
    FD002_TEST_G1_UNCERTAINTY_RISK_628FC
)


if len(
    dev_keys_628fcr2
) != 8367:

    raise RuntimeError(
        "Unexpected DEV key count."
    )


if len(
    dev_g1_628fcr2
) != 8367:

    raise RuntimeError(
        "Unexpected DEV G1 count."
    )


if len(
    test_g1_628fcr2
) != 259:

    raise RuntimeError(
        "Unexpected TEST G1 count."
    )


if not np.isfinite(
    dev_g1_628fcr2
).all():

    raise RuntimeError(
        "DEV G1 contains non-finite values."
    )


if not np.isfinite(
    test_g1_628fcr2
).all():

    raise RuntimeError(
        "TEST G1 contains non-finite values."
    )


# ============================================================
# 4. Verify clean DEV engine structure
# ============================================================

if "unit_id" not in dev_keys_628fcr2.columns:

    raise RuntimeError(
        "DEV window keys do not contain unit_id."
    )


engine_counts_628fcr2 = (
    dev_keys_628fcr2[
        "unit_id"
    ]
    .value_counts()
    .sort_index()
)


n_engines_628fcr2 = int(
    engine_counts_628fcr2.size
)


print(
    "\n--- DEV ENGINE CONTRACT ---"
)

print(
    "DEV windows        :",
    len(
        dev_keys_628fcr2
    )
)

print(
    "DEV engines        :",
    n_engines_628fcr2
)

print(
    "Min windows/engine :",
    int(
        engine_counts_628fcr2.min()
    )
)

print(
    "Median windows     :",
    float(
        engine_counts_628fcr2.median()
    )
)

print(
    "Max windows/engine :",
    int(
        engine_counts_628fcr2.max()
    )
)


if n_engines_628fcr2 != 52:

    raise RuntimeError(
        "Expected exactly 52 DEV engines."
    )


# ============================================================
# 5. Construct equal-engine weights
#
# Each engine contributes total mass = 1 / 52.
#
# If engine i has n_i eligible windows:
#
#     weight(window from engine i)
#       = 1 / (52 * n_i)
#
# This treats every eligible window as a possible
# truncated endpoint while preventing long engines from
# dominating the G2 calibration distribution.
# ============================================================

engine_count_map_628fcr2 = (
    engine_counts_628fcr2
    .to_dict()
)


weights_628fcr2 = (
    dev_keys_628fcr2[
        "unit_id"
    ]
    .map(
        lambda unit_628fcr2:
        1.0
        /
        (
            n_engines_628fcr2
            *
            engine_count_map_628fcr2[
                unit_628fcr2
            ]
        )
    )
    .to_numpy(
        dtype=np.float64
    )
)


weight_sum_628fcr2 = float(
    weights_628fcr2.sum()
)


print(
    "\n--- ENGINE-BALANCED WEIGHTS ---"
)

print(
    "Weight count :",
    len(
        weights_628fcr2
    )
)

print(
    "Weight sum   :",
    weight_sum_628fcr2
)


if not np.isclose(
    weight_sum_628fcr2,
    1.0,
    rtol=0.0,
    atol=1e-12
):

    raise RuntimeError(
        "Engine-balanced weights do not sum to 1."
    )


# Verify each engine contributes equal total mass.
engine_weight_audit_628fcr2 = (
    pd.DataFrame(
        {
            "unit_id":
                dev_keys_628fcr2[
                    "unit_id"
                ].to_numpy(),

            "weight":
                weights_628fcr2,
        }
    )
    .groupby(
        "unit_id"
    )[
        "weight"
    ]
    .sum()
)


expected_engine_mass_628fcr2 = (
    1.0
    /
    n_engines_628fcr2
)


max_engine_mass_error_628fcr2 = float(
    np.max(
        np.abs(
            engine_weight_audit_628fcr2
            .to_numpy()
            -
            expected_engine_mass_628fcr2
        )
    )
)


print(
    "Expected mass/engine :",
    expected_engine_mass_628fcr2
)

print(
    "Max engine-mass error:",
    max_engine_mass_error_628fcr2
)


if max_engine_mass_error_628fcr2 > 1e-12:

    raise RuntimeError(
        "Engines do not have equal total G2 weight."
    )


# ============================================================
# 6. Build sorted weighted pseudo-endpoint reference
# ============================================================

sort_idx_628fcr2 = np.argsort(
    dev_g1_628fcr2,
    kind="mergesort"
)


G2_REFERENCE_VALUES_FD002_628FCR2 = (
    dev_g1_628fcr2[
        sort_idx_628fcr2
    ]
)


G2_REFERENCE_WEIGHTS_FD002_628FCR2 = (
    weights_628fcr2[
        sort_idx_628fcr2
    ]
)


# ============================================================
# 7. Collapse ties
#
# Weighted mid-rank ECDF needs the total probability
# mass for every unique G1 value.
# ============================================================

unique_values_628fcr2, inverse_628fcr2 = (
    np.unique(
        G2_REFERENCE_VALUES_FD002_628FCR2,
        return_inverse=True
    )
)


unique_weights_628fcr2 = np.zeros(
    len(
        unique_values_628fcr2
    ),
    dtype=np.float64
)


np.add.at(
    unique_weights_628fcr2,
    inverse_628fcr2,
    G2_REFERENCE_WEIGHTS_FD002_628FCR2
)


cumulative_weights_628fcr2 = np.cumsum(
    unique_weights_628fcr2
)


left_mass_628fcr2 = (
    cumulative_weights_628fcr2
    -
    unique_weights_628fcr2
)


mid_mass_628fcr2 = (
    left_mass_628fcr2
    +
    0.5
    *
    unique_weights_628fcr2
)


G2_UNIQUE_VALUES_FD002_628FCR2 = (
    unique_values_628fcr2.copy()
)


G2_UNIQUE_WEIGHTS_FD002_628FCR2 = (
    unique_weights_628fcr2.copy()
)


# ============================================================
# 8. Weighted empirical mid-rank transformation
#
# For x:
#
# risk(x) =
#     total weight(reference < x)
#     +
#     0.5 * total weight(reference == x)
#
# Values above the reference maximum map to 1.
# Values below the minimum map to 0.
# ============================================================

def apply_engine_balanced_g2_628fcr2(
    values
):

    x_628fcr2 = vec1d_628fcr2(
        values
    )

    if not np.isfinite(
        x_628fcr2
    ).all():

        raise ValueError(
            "G2 input contains non-finite values."
        )

    result_628fcr2 = np.empty(
        len(
            x_628fcr2
        ),
        dtype=np.float64
    )


    left_idx_628fcr2 = np.searchsorted(
        G2_UNIQUE_VALUES_FD002_628FCR2,
        x_628fcr2,
        side="left"
    )


    right_idx_628fcr2 = np.searchsorted(
        G2_UNIQUE_VALUES_FD002_628FCR2,
        x_628fcr2,
        side="right"
    )


    for i_628fcr2 in range(
        len(
            x_628fcr2
        )
    ):

        li_628fcr2 = int(
            left_idx_628fcr2[
                i_628fcr2
            ]
        )

        ri_628fcr2 = int(
            right_idx_628fcr2[
                i_628fcr2
            ]
        )


        # Exact reference value / tie.
        if ri_628fcr2 > li_628fcr2:

            result_628fcr2[
                i_628fcr2
            ] = mid_mass_628fcr2[
                li_628fcr2
            ]


        # Below reference minimum.
        elif li_628fcr2 == 0:

            result_628fcr2[
                i_628fcr2
            ] = 0.0


        # Above reference maximum.
        elif li_628fcr2 >= len(
            unique_values_628fcr2
        ):

            result_628fcr2[
                i_628fcr2
            ] = 1.0


        # Between two observed reference values.
        else:

            result_628fcr2[
                i_628fcr2
            ] = cumulative_weights_628fcr2[
                li_628fcr2 - 1
            ]


    return result_628fcr2


# ============================================================
# 9. Corrected DEV and label-free TEST final risk
# ============================================================

FD002_DEV_SELECTIVE_RISK_628FCR2 = (
    apply_engine_balanced_g2_628fcr2(
        dev_g1_628fcr2
    )
)


FD002_TEST_SELECTIVE_RISK_628FCR2 = (
    apply_engine_balanced_g2_628fcr2(
        test_g1_628fcr2
    )
)


# ============================================================
# 10. True DEV failure endpoint diagnostic only
# ============================================================

endpoint_positions_628fcr2 = np.asarray(
    DEV_ENDPOINT_POSITIONS_FD002_628FC,
    dtype=int
)


FD002_DEV_TRUE_ENDPOINT_RISK_628FCR2 = (
    FD002_DEV_SELECTIVE_RISK_628FCR2[
        endpoint_positions_628fcr2
    ]
)


# ============================================================
# 11. Numerical integrity
# ============================================================

risk_sets_628fcr2 = {

    "DEV":
        FD002_DEV_SELECTIVE_RISK_628FCR2,

    "TEST":
        FD002_TEST_SELECTIVE_RISK_628FCR2,

    "TRUE_DEV_ENDPOINT":
        FD002_DEV_TRUE_ENDPOINT_RISK_628FCR2,
}


print(
    "\n--- CORRECTED G2 NUMERICAL INTEGRITY ---"
)


for label_628fcr2, arr_628fcr2 in (
    risk_sets_628fcr2.items()
):

    arr_628fcr2 = vec1d_628fcr2(
        arr_628fcr2
    )

    finite_628fcr2 = bool(
        np.isfinite(
            arr_628fcr2
        ).all()
    )

    range_ok_628fcr2 = bool(
        np.all(
            (
                arr_628fcr2
                >=
                0.0
            )
            &
            (
                arr_628fcr2
                <=
                1.0
            )
        )
    )

    print(
        f"{label_628fcr2:<20} "
        f"n={len(arr_628fcr2):<5} "
        f"finite={finite_628fcr2} "
        f"range_ok={range_ok_628fcr2} "
        f"min={np.min(arr_628fcr2):.6f} "
        f"median={np.median(arr_628fcr2):.6f} "
        f"mean={np.mean(arr_628fcr2):.6f} "
        f"max={np.max(arr_628fcr2):.6f}"
    )

    if not finite_628fcr2:

        raise RuntimeError(
            label_628fcr2
            +
            " contains non-finite risk."
        )

    if not range_ok_628fcr2:

        raise RuntimeError(
            label_628fcr2
            +
            " is outside [0,1]."
        )


# ============================================================
# 12. Saturation comparison
# ============================================================

old_dev_g2_628fcr2 = None
old_test_g2_628fcr2 = None


if (
    "DEV_LIFECYCLE_AUDIT_FD002_628FCR1"
    in globals()
):

    old_dev_g2_628fcr2 = (
        DEV_LIFECYCLE_AUDIT_FD002_628FCR1[
            "current_g2_risk"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


if (
    "FD002_TEST_SELECTIVE_RISK_628FC"
    in globals()
):

    old_test_g2_628fcr2 = vec1d_628fcr2(
        FD002_TEST_SELECTIVE_RISK_628FC
    )


def count_saturated_628fcr2(
    x
):

    return int(
        np.sum(
            np.isclose(
                x,
                1.0,
                rtol=0.0,
                atol=1e-15
            )
        )
    )


print(
    "\n--- OLD vs CORRECTED G2 SATURATION ---"
)


if old_dev_g2_628fcr2 is not None:

    print(
        "OLD DEV risk==1 :",
        count_saturated_628fcr2(
            old_dev_g2_628fcr2
        )
    )


print(
    "NEW DEV risk==1 :",
    count_saturated_628fcr2(
        FD002_DEV_SELECTIVE_RISK_628FCR2
    )
)


if old_test_g2_628fcr2 is not None:

    print(
        "OLD TEST risk==1:",
        count_saturated_628fcr2(
            old_test_g2_628fcr2
        )
    )


print(
    "NEW TEST risk==1:",
    count_saturated_628fcr2(
        FD002_TEST_SELECTIVE_RISK_628FCR2
    )
)


# ============================================================
# 13. Correlation diagnostic
# ============================================================

if (
    "Y_DEV_PRIMARY_FD002_628D"
    in globals()
):

    target_628fcr2 = vec1d_628fcr2(
        Y_DEV_PRIMARY_FD002_628D
    )

    if len(
        target_628fcr2
    ) == len(
        FD002_DEV_SELECTIVE_RISK_628FCR2
    ):

        corr_df_628fcr2 = pd.DataFrame(
            {
                "target_rul":
                    target_628fcr2,

                "g1_risk":
                    dev_g1_628fcr2,

                "corrected_g2_risk":
                    FD002_DEV_SELECTIVE_RISK_628FCR2,
            }
        )


        CORRECTED_G2_CORRELATION_628FCR2 = (
            corr_df_628fcr2
            .corr(
                method="spearman"
            )
        )


        print(
            "\n--- CORRECTED DEV SPEARMAN AUDIT ---"
        )

        print(
            CORRECTED_G2_CORRELATION_628FCR2
            .round(4)
            .to_string()
        )


# ============================================================
# 14. Label-free TEST reliability table
# ============================================================

test_meta_628fcr2 = (
    TEST_ENDPOINT_META_FD002_628D
    .copy()
    .reset_index(drop=True)
)


if len(
    test_meta_628fcr2
) != 259:

    raise RuntimeError(
        "Unexpected TEST metadata count."
    )


TEST_RELIABILITY_TABLE_FD002_628FCR2 = (
    test_meta_628fcr2.copy()
)


TEST_RELIABILITY_TABLE_FD002_628FCR2[
    "g1_uncertainty_risk"
] = (
    test_g1_628fcr2
)


TEST_RELIABILITY_TABLE_FD002_628FCR2[
    "selective_risk"
] = (
    FD002_TEST_SELECTIVE_RISK_628FCR2
)


# ============================================================
# 15. Freeze corrected G2 methodology contract
# ============================================================

FD002_G2_REPAIR_CONTRACT_628FCR2 = {

    "dataset":
        "FD002",

    "reason_for_repair":
        (
            "true_failure_endpoint_reference_caused_"
            "severe_clean_population_saturation"
        ),

    "superseded_g2_reference":
        "52_true_failure_DEV_endpoints",

    "new_reference_population":
        "all_clean_DEV_windows_as_possible_truncated_endpoints",

    "engine_balancing":
        True,

    "engine_count":
        52,

    "reference_windows":
        int(
            len(
                dev_g1_628fcr2
            )
        ),

    "engine_weighting_rule":
        "each_engine_equal_total_probability_mass",

    "window_weight_rule":
        "1/(number_of_engines * windows_in_engine)",

    "transform":
        "weighted_empirical_midrank_CDF",

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "corruption_applied":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 16. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R2 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Old G2 reference              : "
    "52 true failure endpoints"
)

print(
    "Old G2 status                 : SUPERSEDED"
)

print(
    "New G2 population             : "
    "all clean DEV candidate endpoints"
)

print(
    "DEV reference windows         :",
    len(
        dev_g1_628fcr2
    )
)

print(
    "DEV reference engines         :",
    n_engines_628fcr2
)

print(
    "Equal total engine weighting  : True"
)

print(
    "G2 transform                  : "
    "weighted empirical mid-rank CDF"
)

print(
    "Corrected DEV risk count      :",
    len(
        FD002_DEV_SELECTIVE_RISK_628FCR2
    )
)

print(
    "Corrected TEST risk count     :",
    len(
        FD002_TEST_SELECTIVE_RISK_628FCR2
    )
)

print(
    "Official FD002 TEST labels read : False"
)

print(
    "TEST scored                     : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Corruption applied              : False"
)

print(
    "Model retrained                 : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.28F-C-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.28F-C-R3
# FD002 CORRECTED RELIABILITY-CALIBRATION CLOSURE
#
# Purpose:
# Freeze the repaired clean reliability-risk chain before
# beginning sensor-degradation experiments.
#
# NO TEST labels
# NO TEST scoring
# NO threshold selection
# NO corruption
# NO model retraining
# NO files written
# ============================================================

import numpy as np


print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R3 — CORRECTED RELIABILITY CALIBRATION CLOSURE\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required repaired objects
# ------------------------------------------------------------

required_628fcr3 = [
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",
    "FD002_TEST_SELECTIVE_RISK_628FCR2",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "FD002_G2_REPAIR_CONTRACT_628FCR2",
]


print("\n--- REQUIRED REPAIRED OBJECTS ---")

for name in required_628fcr3:

    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing repaired object: {name}"
        )


# ------------------------------------------------------------
# 2. Exact population contract
# ------------------------------------------------------------

dev_u = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)

test_u = np.asarray(
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)

dev_g1 = np.asarray(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)

test_g1 = np.asarray(
    FD002_TEST_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)

dev_final = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)

test_final = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


expected_lengths = {
    "DEV adjusted uncertainty": (len(dev_u), 8367),
    "TEST adjusted uncertainty": (len(test_u), 259),
    "DEV G1 risk": (len(dev_g1), 8367),
    "TEST G1 risk": (len(test_g1), 259),
    "DEV final risk": (len(dev_final), 8367),
    "TEST final risk": (len(test_final), 259),
}


print("\n--- POPULATION CONTRACT ---")

for label, (observed, expected) in expected_lengths.items():

    ok = observed == expected

    print(
        f"{label:<30}: "
        f"{observed} | expected={expected} | ok={ok}"
    )

    if not ok:
        raise RuntimeError(
            f"Population mismatch: {label}"
        )


# ------------------------------------------------------------
# 3. Numerical integrity
# ------------------------------------------------------------

print("\n--- NUMERICAL INTEGRITY ---")

risk_objects = {
    "DEV adjusted-U": dev_u,
    "TEST adjusted-U": test_u,
    "DEV G1": dev_g1,
    "TEST G1": test_g1,
    "DEV final": dev_final,
    "TEST final": test_final,
}


for label, arr in risk_objects.items():

    finite = bool(
        np.isfinite(arr).all()
    )

    within = bool(
        np.all(
            (arr >= 0.0)
            &
            (arr <= 1.0)
        )
    )

    print(
        f"{label:<18} "
        f"finite={finite} "
        f"within_[0,1]={within} "
        f"min={arr.min():.6f} "
        f"median={np.median(arr):.6f} "
        f"mean={arr.mean():.6f} "
        f"max={arr.max():.6f}"
    )

    if not finite or not within:
        raise RuntimeError(
            f"Numerical integrity failed: {label}"
        )


# ------------------------------------------------------------
# 4. Engine-balanced G2 reference integrity
# ------------------------------------------------------------

g2_values = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

g2_weights = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


print("\n--- G2 REFERENCE CONTRACT ---")

print(
    "Reference values :",
    len(g2_values)
)

print(
    "Reference weights:",
    len(g2_weights)
)

print(
    "Weight sum       :",
    float(g2_weights.sum())
)

print(
    "Sorted values    :",
    bool(
        np.all(
            np.diff(g2_values) >= 0.0
        )
    )
)


if len(g2_values) != 8367:
    raise RuntimeError(
        "Corrected G2 reference length must be 8367."
    )

if len(g2_weights) != 8367:
    raise RuntimeError(
        "Corrected G2 weight length must be 8367."
    )

if not np.isclose(
    g2_weights.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12
):
    raise RuntimeError(
        "Corrected G2 weights do not sum to 1."
    )


# ------------------------------------------------------------
# 5. Saturation closure
# ------------------------------------------------------------

dev_one = int(
    np.sum(
        np.isclose(
            dev_final,
            1.0,
            rtol=0.0,
            atol=1e-15
        )
    )
)

test_one = int(
    np.sum(
        np.isclose(
            test_final,
            1.0,
            rtol=0.0,
            atol=1e-15
        )
    )
)


print("\n--- SATURATION CLOSURE ---")

print(
    "DEV final risk == 1 :",
    dev_one
)

print(
    "TEST final risk == 1:",
    test_one
)


if dev_one != 0:
    raise RuntimeError(
        "Unexpected DEV saturation remains."
    )

if test_one != 0:
    raise RuntimeError(
        "Unexpected TEST saturation remains."
    )


# ------------------------------------------------------------
# 6. Ordering contract
#
# Since G2 is a monotonic empirical calibration of G1,
# rank ordering should be preserved.
# ------------------------------------------------------------

dev_order_g1 = np.argsort(
    dev_g1,
    kind="mergesort"
)

dev_order_g2 = np.argsort(
    dev_final,
    kind="mergesort"
)


ordering_exact = bool(
    np.array_equal(
        dev_order_g1,
        dev_order_g2
    )
)


print("\n--- MONOTONICITY CONTRACT ---")

print(
    "DEV G1/G2 rank ordering exact:",
    ordering_exact
)


# Exact equality can be affected by ties, so also verify
# pairwise monotonicity through sorted values.

sorted_final = dev_final[
    dev_order_g1
]

monotonic_ok = bool(
    np.all(
        np.diff(
            sorted_final
        )
        >=
        -1e-14
    )
)


print(
    "G2 monotonic in G1          :",
    monotonic_ok
)


if not monotonic_ok:
    raise RuntimeError(
        "Corrected G2 is not monotonic in G1."
    )


# ------------------------------------------------------------
# 7. Freeze final clean reliability contract
# ------------------------------------------------------------

FD002_CLEAN_RELIABILITY_CONTRACT_628FCR3 = {

    "dataset":
        "FD002",

    "uncertainty_recipe":
        "det_pred_pct_b5",

    "phase_reference":
        "FD002_FIT_only",

    "phase_bins":
        5,

    "g1_reference":
        "all_clean_FD002_DEV_windows",

    "g1_transform":
        "right_empirical_CDF",

    "g2_reference":
        (
            "all_clean_FD002_DEV_windows_as_"
            "possible_truncated_endpoints"
        ),

    "g2_engine_balanced":
        True,

    "g2_transform":
        "weighted_empirical_midrank_CDF",

    "old_true_failure_endpoint_g2":
        "superseded",

    "dev_windows":
        8367,

    "dev_engines":
        52,

    "test_engines":
        259,

    "threshold_status":
        "NOT_SELECTED",

    "corruption_status":
        "NOT_APPLIED",

    "official_test_labels_used":
        False,

    "test_scored":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.28F-C-R3 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Phase adjustment          : FIT-calibrated"
)

print(
    "G1 calibration            : clean DEV windows"
)

print(
    "G2 calibration            : engine-balanced "
    "clean DEV candidate endpoints"
)

print(
    "Old failure-endpoint G2   : SUPERSEDED"
)

print(
    "DEV saturation remaining  :",
    dev_one
)

print(
    "TEST saturation remaining :",
    test_one
)

print(
    "G2 monotonic in G1        :",
    monotonic_ok
)

print(
    "FD002 clean reliability chain frozen : True"
)

print(
    "Official FD002 TEST labels read      : False"
)

print(
    "TEST scored                          : False"
)

print(
    "Threshold selected                   : False"
)

print(
    "Corruption applied                   : False"
)

print(
    "Model retrained                      : False"
)

print(
    "Files written                        : False"
)

print(
    "\n=== END STEP 6.28F-C-R3 ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-A
# RECOVER FROZEN FD001 SENSOR-DEGRADATION PROTOCOL
#
# PURPOSE
# -------
# Recover the exact corruption definitions used previously
# for FD001 before reproducing them on FD002.
#
# We intentionally DO NOT guess severity values or formulas.
#
# Searches existing FD001 notebooks for:
#   noise
#   bias
#   drift
#   dropout / missing
#   stuck sensor
#   severity
#   corruption sensor selection
#   random seeds
#
# IMPORTANT
# ---------
# - Source-code inspection only
# - No notebook cells executed
# - No model training
# - No TEST labels read
# - No TEST scoring
# - No corruption applied
# - No files written
# ============================================================

from pathlib import Path
import json
import re


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-A — FD001 DEGRADATION-PROTOCOL RECOVERY\n"
    "============================================================"
)


# ============================================================
# 1. Resolve project root
# ============================================================

cwd_628ga = Path.cwd()


if (cwd_628ga / "03_notebooks").exists():

    PROJECT_ROOT_628GA = cwd_628ga

elif (cwd_628ga.parent / "03_notebooks").exists():

    PROJECT_ROOT_628GA = cwd_628ga.parent

else:

    raise FileNotFoundError(
        "Could not locate project root containing 03_notebooks."
    )


NOTEBOOK_DIR_628GA = (
    PROJECT_ROOT_628GA
    /
    "03_notebooks"
)


print(
    "\n--- PROJECT CONTRACT ---"
)

print(
    "Project root      :",
    PROJECT_ROOT_628GA
)

print(
    "Notebook directory:",
    NOTEBOOK_DIR_628GA
)

print(
    "Directory exists  :",
    NOTEBOOK_DIR_628GA.exists()
)


# ============================================================
# 2. FD001 notebooks to inspect
# ============================================================

candidate_names_628ga = [
    "04_sensor_degradation_fd001.ipynb",
    "05_degradation_impact_fd001.ipynb",
    "06_reliability_signals_fd001.ipynb",
]


candidate_paths_628ga = [
    NOTEBOOK_DIR_628GA / name_628ga
    for name_628ga in candidate_names_628ga
]


print(
    "\n--- FD001 SOURCE NOTEBOOKS ---"
)


for path_628ga in candidate_paths_628ga:

    print(
        f"{path_628ga.name:<40}: "
        f"{path_628ga.exists()}"
    )


existing_paths_628ga = [
    path_628ga
    for path_628ga in candidate_paths_628ga
    if path_628ga.exists()
]


if not existing_paths_628ga:

    raise RuntimeError(
        "No FD001 source notebooks were found."
    )


# ============================================================
# 3. Search vocabulary
# ============================================================

search_terms_628ga = [
    "noise",
    "gaussian",
    "bias",
    "drift",
    "dropout",
    "missing",
    "stuck",
    "severity",
    "corrupt",
    "degrad",
    "sensor",
    "random",
    "seed",
]


pattern_628ga = re.compile(
    "|".join(
        re.escape(term_628ga)
        for term_628ga in search_terms_628ga
    ),
    flags=re.IGNORECASE
)


# ============================================================
# 4. Read code cells and recover relevant candidates
# ============================================================

matches_628ga = []


for notebook_path_628ga in existing_paths_628ga:

    with open(
        notebook_path_628ga,
        "r",
        encoding="utf-8"
    ) as f_628ga:

        notebook_json_628ga = json.load(
            f_628ga
        )


    cells_628ga = notebook_json_628ga.get(
        "cells",
        []
    )


    for cell_index_628ga, cell_628ga in enumerate(
        cells_628ga
    ):

        if cell_628ga.get(
            "cell_type"
        ) != "code":

            continue


        source_628ga = "".join(
            cell_628ga.get(
                "source",
                []
            )
        )


        if not source_628ga.strip():

            continue


        term_hits_628ga = sorted(
            {
                match_628ga.group(0).lower()
                for match_628ga
                in pattern_628ga.finditer(
                    source_628ga
                )
            }
        )


        if not term_hits_628ga:

            continue


        # Extra relevance score:
        # corruption-family terms are more important
        # than generic words such as sensor/seed.
        important_terms_628ga = {
            "noise",
            "gaussian",
            "bias",
            "drift",
            "dropout",
            "missing",
            "stuck",
            "severity",
            "corrupt",
            "degrad",
        }


        important_count_628ga = sum(
            term_628ga in important_terms_628ga
            for term_628ga in term_hits_628ga
        )


        if important_count_628ga == 0:

            continue


        matches_628ga.append(
            {
                "notebook":
                    notebook_path_628ga.name,

                "cell_index":
                    int(
                        cell_index_628ga
                    ),

                "hits":
                    term_hits_628ga,

                "important_hits":
                    int(
                        important_count_628ga
                    ),

                "source":
                    source_628ga,
            }
        )


# ============================================================
# 5. Sort most relevant cells first
# ============================================================

matches_628ga = sorted(
    matches_628ga,
    key=lambda row_628ga: (
        -row_628ga[
            "important_hits"
        ],
        row_628ga[
            "notebook"
        ],
        row_628ga[
            "cell_index"
        ],
    )
)


print(
    "\n--- RECOVERED RELEVANT CODE CELLS ---"
)

print(
    "Relevant cells found:",
    len(
        matches_628ga
    )
)


# ============================================================
# 6. Print complete relevant source cells
#
# Limit prevents VS Code output from becoming unmanageable.
# Usually the corruption definition is within the first
# several high-scoring cells.
# ============================================================

MAX_CELLS_TO_PRINT_628GA = 20


for rank_628ga, row_628ga in enumerate(
    matches_628ga[
        :MAX_CELLS_TO_PRINT_628GA
    ],
    start=1
):

    print(
        "\n"
        "============================================================"
    )

    print(
        f"MATCH {rank_628ga:02d}"
    )

    print(
        "Notebook :",
        row_628ga[
            "notebook"
        ]
    )

    print(
        "Cell     :",
        row_628ga[
            "cell_index"
        ]
    )

    print(
        "Hits     :",
        row_628ga[
            "hits"
        ]
    )

    print(
        "------------------------------------------------------------"
    )

    print(
        row_628ga[
            "source"
        ]
    )


# ============================================================
# 7. Compact variable-line extraction
#
# This additionally extracts individual lines likely to
# contain the actual frozen corruption parameters.
# ============================================================

parameter_keywords_628ga = [
    "severity",
    "sigma",
    "noise",
    "bias",
    "drift",
    "dropout",
    "missing",
    "stuck",
    "seed",
    "sensor",
    "fraction",
    "scale",
    "level",
    "rate",
]


parameter_pattern_628ga = re.compile(
    "|".join(
        re.escape(term_628ga)
        for term_628ga
        in parameter_keywords_628ga
    ),
    flags=re.IGNORECASE
)


parameter_lines_628ga = []


for row_628ga in matches_628ga:

    for line_number_628ga, line_628ga in enumerate(
        row_628ga[
            "source"
        ].splitlines(),
        start=1
    ):

        stripped_628ga = line_628ga.strip()


        if not stripped_628ga:

            continue


        if parameter_pattern_628ga.search(
            stripped_628ga
        ):

            parameter_lines_628ga.append(
                {
                    "notebook":
                        row_628ga[
                            "notebook"
                        ],

                    "cell":
                        row_628ga[
                            "cell_index"
                        ],

                    "line":
                        line_number_628ga,

                    "text":
                        stripped_628ga,
                }
            )


print(
    "\n"
    "============================================================\n"
    "COMPACT PARAMETER-LINE REVIEW\n"
    "============================================================"
)


for item_628ga in parameter_lines_628ga[
    :150
]:

    print(
        f"{item_628ga['notebook']} "
        f"| cell={item_628ga['cell']:03d} "
        f"| line={item_628ga['line']:03d} "
        f"| {item_628ga['text']}"
    )


# ============================================================
# 8. Freeze recovery audit only
# ============================================================

FD001_DEGRADATION_RECOVERY_AUDIT_628GA = {

    "project_root":
        str(
            PROJECT_ROOT_628GA
        ),

    "source_notebooks":
        [
            path_628ga.name
            for path_628ga
            in existing_paths_628ga
        ],

    "relevant_code_cells":
        int(
            len(
                matches_628ga
            )
        ),

    "parameter_lines":
        int(
            len(
                parameter_lines_628ga
            )
        ),

    "old_notebook_cells_executed":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "corruption_applied":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 9. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 notebooks inspected :",
    len(
        existing_paths_628ga
    )
)

print(
    "Relevant code cells found :",
    len(
        matches_628ga
    )
)

print(
    "Candidate parameter lines :",
    len(
        parameter_lines_628ga
    )
)

print(
    "Old cells executed         : False"
)

print(
    "Official TEST labels read  : False"
)

print(
    "TEST scored                : False"
)

print(
    "Corruption applied         : False"
)

print(
    "Model retrained            : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.28G-A ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-A-R1
# EXACT FROZEN FD001 CORRUPTION-PARAMETER RECOVERY
#
# PURPOSE
# -------
# Read the five already-frozen FD001 scenario/result manifests
# and recover ONLY the information needed to reproduce the
# corruption protocol on FD002:
#
#   - retained sensors
#   - severity labels
#   - numerical alpha values
#   - directions
#   - requested fractions
#   - seeds
#   - scenario counts
#
# IMPORTANT
# ---------
# - READ ONLY
# - No corruption applied
# - No model execution
# - No TEST labels read
# - No TEST scoring
# - No files written
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-A-R1 — EXACT FD001 CORRUPTION PARAMETERS\n"
    "============================================================"
)


# ============================================================
# 1. Resolve project root
# ============================================================

cwd_628gar1 = Path.cwd()

if (cwd_628gar1 / "06_results").exists():

    PROJECT_ROOT_628GAR1 = cwd_628gar1

elif (cwd_628gar1.parent / "06_results").exists():

    PROJECT_ROOT_628GAR1 = cwd_628gar1.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


RESULT_DIR_628GAR1 = (
    PROJECT_ROOT_628GAR1
    / "06_results"
    / "step6_reliability_signals"
)


print(
    "\n--- PROJECT CONTRACT ---"
)

print(
    "Project root    :",
    PROJECT_ROOT_628GAR1
)

print(
    "Result directory:",
    RESULT_DIR_628GAR1
)

print(
    "Directory exists:",
    RESULT_DIR_628GAR1.exists()
)


# ============================================================
# 2. Exact frozen FD001 manifest/result files
# ============================================================

FILE_SPECS_628GAR1 = {

    "bias": {
        "filename":
            "fd001_pca_bias_response_90_scenarios.csv",
        "expected_rows": 90,
    },

    "noise": {
        "filename":
            "fd001_pca_noise_response_45_scenarios.csv",
        "expected_rows": 45,
    },

    "drift": {
        "filename":
            "fd001_pca_drift_response_90_scenarios.csv",
        "expected_rows": 90,
    },

    "dropout": {
        "filename":
            "fd001_pca_dropout_response_45_scenarios.csv",
        "expected_rows": 45,
    },

    "stuck": {
        "filename":
            "fd001_pca_stuck_response_45_scenarios.csv",
        "expected_rows": 45,
    },
}


# ============================================================
# 3. Load manifests READ ONLY
# ============================================================

manifests_628gar1 = {}


print(
    "\n--- FROZEN FILE CHECK ---"
)


for family_628gar1, spec_628gar1 in (
    FILE_SPECS_628GAR1.items()
):

    path_628gar1 = (
        RESULT_DIR_628GAR1
        / spec_628gar1["filename"]
    )

    exists_628gar1 = path_628gar1.exists()

    print(
        f"{family_628gar1:<10} "
        f"exists={exists_628gar1} "
        f"| {path_628gar1.name}"
    )

    if not exists_628gar1:

        raise FileNotFoundError(
            f"Missing frozen FD001 file: "
            f"{path_628gar1}"
        )

    df_628gar1 = pd.read_csv(
        path_628gar1
    )

    expected_rows_628gar1 = int(
        spec_628gar1["expected_rows"]
    )

    if len(df_628gar1) != expected_rows_628gar1:

        raise RuntimeError(
            f"{family_628gar1}: expected "
            f"{expected_rows_628gar1} rows, "
            f"found {len(df_628gar1)}."
        )

    manifests_628gar1[
        family_628gar1
    ] = df_628gar1.copy()


# ============================================================
# 4. Helper for exact sorted unique numerical values
# ============================================================

def numeric_unique_628gar1(
    df,
    column
):

    if column not in df.columns:
        return []

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    values = values[
        values.notna()
    ].to_numpy(
        dtype=np.float64
    )

    if len(values) == 0:
        return []

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Non-finite values in {column}."
        )

    return sorted(
        np.unique(
            values
        ).tolist()
    )


# ============================================================
# 5. Structural parameter recovery
# ============================================================

summary_rows_628gar1 = []

sensor_sets_628gar1 = {}


for family_628gar1, df_628gar1 in (
    manifests_628gar1.items()
):

    required_columns_628gar1 = [
        "sensor",
        "severity",
    ]

    missing_628gar1 = [
        column_628gar1
        for column_628gar1
        in required_columns_628gar1
        if column_628gar1
        not in df_628gar1.columns
    ]

    if missing_628gar1:

        raise RuntimeError(
            f"{family_628gar1}: missing columns "
            f"{missing_628gar1}"
        )


    sensors_628gar1 = sorted(
        df_628gar1[
            "sensor"
        ]
        .astype(str)
        .unique()
        .tolist()
    )


    severities_628gar1 = sorted(
        df_628gar1[
            "severity"
        ]
        .astype(str)
        .str.lower()
        .unique()
        .tolist()
    )


    sensor_sets_628gar1[
        family_628gar1
    ] = set(
        sensors_628gar1
    )


    summary_rows_628gar1.append(
        {
            "family":
                family_628gar1,

            "rows":
                len(df_628gar1),

            "sensors":
                len(sensors_628gar1),

            "severity_labels":
                severities_628gar1,

            "alpha_values":
                numeric_unique_628gar1(
                    df_628gar1,
                    "alpha"
                ),

            "direction_values":
                numeric_unique_628gar1(
                    df_628gar1,
                    "direction"
                ),

            "requested_fractions":
                numeric_unique_628gar1(
                    df_628gar1,
                    "requested_fraction"
                ),

            "seed_values":
                numeric_unique_628gar1(
                    df_628gar1,
                    "seed"
                ),
        }
    )


PARAMETER_SUMMARY_FD001_628GAR1 = (
    pd.DataFrame(
        summary_rows_628gar1
    )
)


# ============================================================
# 6. Verify common 15-sensor contract
# ============================================================

reference_family_628gar1 = "bias"

REFERENCE_SENSOR_SET_628GAR1 = (
    sensor_sets_628gar1[
        reference_family_628gar1
    ]
)


sensor_sets_equal_628gar1 = all(

    sensor_set_628gar1
    ==
    REFERENCE_SENSOR_SET_628GAR1

    for sensor_set_628gar1
    in sensor_sets_628gar1.values()
)


if not sensor_sets_equal_628gar1:

    raise RuntimeError(
        "FD001 corruption families do not use "
        "the same sensor set."
    )


if len(
    REFERENCE_SENSOR_SET_628GAR1
) != 15:

    raise RuntimeError(
        "Expected exactly 15 frozen FD001 sensors."
    )


RETAINED_SENSORS_FD001_628GAR1 = sorted(
    REFERENCE_SENSOR_SET_628GAR1
)


# ============================================================
# 7. Methodology integrity
# ============================================================

EXPECTED_SEVERITIES_628GAR1 = {
    "low",
    "medium",
    "high",
}


for family_628gar1, df_628gar1 in (
    manifests_628gar1.items()
):

    observed_628gar1 = set(
        df_628gar1[
            "severity"
        ]
        .astype(str)
        .str.lower()
        .unique()
    )

    if observed_628gar1 != (
        EXPECTED_SEVERITIES_628GAR1
    ):

        raise RuntimeError(
            f"{family_628gar1}: unexpected "
            f"severity labels "
            f"{sorted(observed_628gar1)}"
        )


# Bias and drift must preserve ± direction.
for family_628gar1 in [
    "bias",
    "drift",
]:

    directions_628gar1 = set(
        numeric_unique_628gar1(
            manifests_628gar1[
                family_628gar1
            ],
            "direction"
        )
    )

    if directions_628gar1 != {
        -1.0,
        1.0,
    }:

        raise RuntimeError(
            f"{family_628gar1}: expected "
            f"directions [-1, +1], found "
            f"{sorted(directions_628gar1)}"
        )


# Bias / noise / drift require numerical alpha.
for family_628gar1 in [
    "bias",
    "noise",
    "drift",
]:

    alpha_628gar1 = numeric_unique_628gar1(
        manifests_628gar1[
            family_628gar1
        ],
        "alpha"
    )

    if len(
        alpha_628gar1
    ) != 3:

        raise RuntimeError(
            f"{family_628gar1}: expected "
            f"three frozen alpha values, "
            f"found {alpha_628gar1}"
        )


# Dropout / stuck require numerical fraction.
for family_628gar1 in [
    "dropout",
    "stuck",
]:

    fractions_628gar1 = (
        numeric_unique_628gar1(
            manifests_628gar1[
                family_628gar1
            ],
            "requested_fraction"
        )
    )

    if len(
        fractions_628gar1
    ) != 3:

        raise RuntimeError(
            f"{family_628gar1}: expected "
            f"three frozen fractions, "
            f"found {fractions_628gar1}"
        )


# ============================================================
# 8. Compact display — THIS is the important output
# ============================================================

print(
    "\n"
    "============================================================\n"
    "EXACT FD001 FROZEN PARAMETER SUMMARY\n"
    "============================================================"
)


for _, row_628gar1 in (
    PARAMETER_SUMMARY_FD001_628GAR1.iterrows()
):

    print(
        f"\nFamily              : "
        f"{row_628gar1['family']}"
    )

    print(
        f"Scenario rows       : "
        f"{row_628gar1['rows']}"
    )

    print(
        f"Sensors             : "
        f"{row_628gar1['sensors']}"
    )

    print(
        f"Severity labels     : "
        f"{row_628gar1['severity_labels']}"
    )

    print(
        f"Alpha values        : "
        f"{row_628gar1['alpha_values']}"
    )

    print(
        f"Direction values    : "
        f"{row_628gar1['direction_values']}"
    )

    print(
        f"Requested fractions : "
        f"{row_628gar1['requested_fractions']}"
    )

    print(
        f"Seed values         : "
        f"{row_628gar1['seed_values']}"
    )


print(
    "\n--- EXACT RETAINED SENSOR SET ---"
)

print(
    RETAINED_SENSORS_FD001_628GAR1
)


# ============================================================
# 9. Freeze recovered protocol in memory
# ============================================================

FD001_CORRUPTION_PROTOCOL_628GAR1 = {

    "families": [
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ],

    "retained_sensors":
        RETAINED_SENSORS_FD001_628GAR1,

    "bias_alpha":
        numeric_unique_628gar1(
            manifests_628gar1["bias"],
            "alpha"
        ),

    "noise_alpha":
        numeric_unique_628gar1(
            manifests_628gar1["noise"],
            "alpha"
        ),

    "drift_alpha":
        numeric_unique_628gar1(
            manifests_628gar1["drift"],
            "alpha"
        ),

    "bias_directions":
        numeric_unique_628gar1(
            manifests_628gar1["bias"],
            "direction"
        ),

    "drift_directions":
        numeric_unique_628gar1(
            manifests_628gar1["drift"],
            "direction"
        ),

    "dropout_fractions":
        numeric_unique_628gar1(
            manifests_628gar1["dropout"],
            "requested_fraction"
        ),

    "stuck_fractions":
        numeric_unique_628gar1(
            manifests_628gar1["stuck"],
            "requested_fraction"
        ),

    "noise_seeds":
        numeric_unique_628gar1(
            manifests_628gar1["noise"],
            "seed"
        ),
}


# ============================================================
# 10. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-A-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Corruption families       : 5"
)

print(
    "Frozen scenarios          : 315"
)

print(
    "Common retained sensors   :",
    len(
        RETAINED_SENSORS_FD001_628GAR1
    )
)

print(
    "Sensor sets identical     :",
    sensor_sets_equal_628gar1
)

print(
    "Exact numerical levels recovered: True"
)

print(
    "Corruption applied        : False"
)

print(
    "Official TEST labels read : False"
)

print(
    "TEST scored               : False"
)

print(
    "Model retrained           : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.28G-A-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-B
# FD002 CORRUPTION-TRANSFER CONTRACT
#
# PURPOSE
# -------
# 1. Recover exact FD001 severity -> numerical parameter mapping.
# 2. Verify the FD001 sensor set against FD002 model sensors.
# 3. Attach FD002 FIT operating-regime IDs to RAW FIT data.
# 4. Build FIT-only condition-adjusted sensor scale references.
# 5. Freeze the methodology that will later generate FD002
#    corruption scenarios.
#
# IMPORTANT
# ---------
# - No corruption is applied.
# - No DEV data are used to fit corruption scales.
# - No official TEST labels are read.
# - No TEST scoring.
# - No model retraining.
# - No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-B — FD002 CORRUPTION-TRANSFER CONTRACT\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628gb = [
    "manifests_628gar1",
    "FD001_CORRUPTION_PROTOCOL_628GAR1",
    "FD002_FIT_TARGETED_628B",
    "FD002_FIT_NORMALIZED_628C",
    "FD002_MODEL_FEATURES_628C",
]


print(
    "\n--- REQUIRED OBJECTS ---"
)


missing_628gb = []

for name_628gb in required_628gb:

    available_628gb = (
        name_628gb in globals()
    )

    print(
        f"{name_628gb:<35}: "
        f"{available_628gb}"
    )

    if not available_628gb:
        missing_628gb.append(
            name_628gb
        )


if missing_628gb:

    raise RuntimeError(
        "Missing required objects: "
        + str(
            missing_628gb
        )
    )


# ============================================================
# 2. RECOVER EXACT SEVERITY -> PARAMETER MAPPING
# ============================================================

def severity_parameter_map_628gb(
    df,
    parameter_column
):

    required_columns_628gb = [
        "severity",
        parameter_column,
    ]

    missing_columns_628gb = [
        column_628gb
        for column_628gb
        in required_columns_628gb
        if column_628gb
        not in df.columns
    ]

    if missing_columns_628gb:

        raise RuntimeError(
            f"Missing mapping columns: "
            f"{missing_columns_628gb}"
        )


    temp_628gb = (
        df[
            [
                "severity",
                parameter_column,
            ]
        ]
        .copy()
    )


    temp_628gb[
        "severity"
    ] = (
        temp_628gb[
            "severity"
        ]
        .astype(str)
        .str.lower()
    )


    temp_628gb[
        parameter_column
    ] = pd.to_numeric(
        temp_628gb[
            parameter_column
        ],
        errors="coerce"
    )


    if not np.isfinite(
        temp_628gb[
            parameter_column
        ].to_numpy(
            dtype=np.float64
        )
    ).all():

        raise RuntimeError(
            f"Non-finite values in "
            f"{parameter_column}."
        )


    uniqueness_628gb = (
        temp_628gb
        .groupby(
            "severity"
        )[
            parameter_column
        ]
        .nunique()
    )


    if not (
        uniqueness_628gb
        == 1
    ).all():

        raise RuntimeError(
            f"Severity does not map uniquely "
            f"to {parameter_column}."
        )


    mapping_628gb = (
        temp_628gb
        .groupby(
            "severity"
        )[
            parameter_column
        ]
        .first()
        .to_dict()
    )


    expected_labels_628gb = {
        "low",
        "medium",
        "high",
    }


    if set(
        mapping_628gb.keys()
    ) != expected_labels_628gb:

        raise RuntimeError(
            "Unexpected severity labels: "
            + str(
                sorted(
                    mapping_628gb.keys()
                )
            )
        )


    return {
        key_628gb:
            float(
                value_628gb
            )
        for (
            key_628gb,
            value_628gb
        )
        in mapping_628gb.items()
    }


BIAS_SEVERITY_MAP_628GB = (
    severity_parameter_map_628gb(
        manifests_628gar1[
            "bias"
        ],
        "alpha"
    )
)


NOISE_SEVERITY_MAP_628GB = (
    severity_parameter_map_628gb(
        manifests_628gar1[
            "noise"
        ],
        "alpha"
    )
)


DRIFT_SEVERITY_MAP_628GB = (
    severity_parameter_map_628gb(
        manifests_628gar1[
            "drift"
        ],
        "alpha"
    )
)


DROPOUT_SEVERITY_MAP_628GB = (
    severity_parameter_map_628gb(
        manifests_628gar1[
            "dropout"
        ],
        "requested_fraction"
    )
)


STUCK_SEVERITY_MAP_628GB = (
    severity_parameter_map_628gb(
        manifests_628gar1[
            "stuck"
        ],
        "requested_fraction"
    )
)


print(
    "\n--- EXACT FD001 SEVERITY MAPPING ---"
)

print(
    "Bias    :",
    BIAS_SEVERITY_MAP_628GB
)

print(
    "Noise   :",
    NOISE_SEVERITY_MAP_628GB
)

print(
    "Drift   :",
    DRIFT_SEVERITY_MAP_628GB
)

print(
    "Dropout :",
    DROPOUT_SEVERITY_MAP_628GB
)

print(
    "Stuck   :",
    STUCK_SEVERITY_MAP_628GB
)


# ============================================================
# 3. RECOVER FD002 MODEL SENSOR ORDER
#
# IMPORTANT:
# use FD002's frozen model order,
# NOT alphabetically sorted FD001 sensor names.
# ============================================================

FD002_MODEL_SENSOR_FEATURES_628GB = [

    feature_628gb

    for feature_628gb
    in FD002_MODEL_FEATURES_628C

    if feature_628gb.startswith(
        "sensor_"
    )
]


FD002_MODEL_SENSORS_628GB = [

    feature_628gb[
        :-2
    ]

    if feature_628gb.endswith(
        "_z"
    )

    else feature_628gb

    for feature_628gb
    in FD002_MODEL_SENSOR_FEATURES_628GB
]


print(
    "\n--- FD002 MODEL SENSOR ORDER ---"
)

print(
    FD002_MODEL_SENSORS_628GB
)

print(
    "Sensor count:",
    len(
        FD002_MODEL_SENSORS_628GB
    )
)


if len(
    FD002_MODEL_SENSORS_628GB
) != 15:

    raise RuntimeError(
        "Expected exactly 15 FD002 "
        "model sensors."
    )


# ============================================================
# 4. VERIFY FD001 AND FD002 SENSOR SETS
# ============================================================

FD001_SENSOR_SET_628GB = set(
    FD001_CORRUPTION_PROTOCOL_628GAR1[
        "retained_sensors"
    ]
)


FD002_SENSOR_SET_628GB = set(
    FD002_MODEL_SENSORS_628GB
)


sensor_sets_match_628gb = bool(
    FD001_SENSOR_SET_628GB
    ==
    FD002_SENSOR_SET_628GB
)


print(
    "\n--- SENSOR TRANSFER CONTRACT ---"
)

print(
    "FD001 sensors:",
    len(
        FD001_SENSOR_SET_628GB
    )
)

print(
    "FD002 sensors:",
    len(
        FD002_SENSOR_SET_628GB
    )
)

print(
    "Sensor sets exactly match:",
    sensor_sets_match_628gb
)


if not sensor_sets_match_628gb:

    print(
        "Only FD001:",
        sorted(
            FD001_SENSOR_SET_628GB
            -
            FD002_SENSOR_SET_628GB
        )
    )

    print(
        "Only FD002:",
        sorted(
            FD002_SENSOR_SET_628GB
            -
            FD001_SENSOR_SET_628GB
        )
    )

    raise RuntimeError(
        "FD001/FD002 retained sensor "
        "sets do not match."
    )


# ============================================================
# 5. RAW FIT + REGIME ALIGNMENT
# ============================================================

raw_fit_628gb = (
    FD002_FIT_TARGETED_628B
    .copy()
)


normalized_fit_628gb = (
    FD002_FIT_NORMALIZED_628C
    .copy()
)


required_key_columns_628gb = [
    "unit_id",
    "cycle",
]


for column_628gb in required_key_columns_628gb:

    if column_628gb not in raw_fit_628gb.columns:

        raise RuntimeError(
            f"{column_628gb} missing "
            f"from raw FIT."
        )

    if column_628gb not in normalized_fit_628gb.columns:

        raise RuntimeError(
            f"{column_628gb} missing "
            f"from normalized FIT."
        )


if "regime_id" not in normalized_fit_628gb.columns:

    raise RuntimeError(
        "regime_id missing from "
        "FD002_FIT_NORMALIZED_628C."
    )


for sensor_628gb in (
    FD002_MODEL_SENSORS_628GB
):

    if sensor_628gb not in raw_fit_628gb.columns:

        raise RuntimeError(
            f"{sensor_628gb} missing "
            f"from raw FD002 FIT."
        )


raw_keys_628gb = (
    raw_fit_628gb[
        required_key_columns_628gb
    ]
    .reset_index(
        drop=True
    )
)


norm_keys_628gb = (
    normalized_fit_628gb[
        required_key_columns_628gb
    ]
    .reset_index(
        drop=True
    )
)


exact_row_alignment_628gb = (
    raw_keys_628gb.equals(
        norm_keys_628gb
    )
)


print(
    "\n--- FIT / REGIME ALIGNMENT ---"
)

print(
    "RAW FIT rows       :",
    len(
        raw_fit_628gb
    )
)

print(
    "Normalized FIT rows:",
    len(
        normalized_fit_628gb
    )
)

print(
    "Exact key alignment:",
    exact_row_alignment_628gb
)


if not exact_row_alignment_628gb:

    raise RuntimeError(
        "RAW and normalized FIT rows "
        "are not exactly aligned."
    )


FD002_FIT_CORRUPTION_REFERENCE_628GB = (
    raw_fit_628gb.copy()
)


FD002_FIT_CORRUPTION_REFERENCE_628GB[
    "regime_id"
] = (
    normalized_fit_628gb[
        "regime_id"
    ]
    .to_numpy()
)


# ============================================================
# 6. REGIME STRUCTURE AUDIT
# ============================================================

regime_counts_628gb = (
    FD002_FIT_CORRUPTION_REFERENCE_628GB[
        "regime_id"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\n--- FD002 FIT REGIME COUNTS ---"
)

print(
    regime_counts_628gb
)


n_regimes_628gb = int(
    regime_counts_628gb.shape[0]
)


print(
    "Operating regimes:",
    n_regimes_628gb
)


if n_regimes_628gb != 6:

    raise RuntimeError(
        "Expected exactly 6 FD002 "
        "operating regimes."
    )


# ============================================================
# 7. PER-REGIME SENSOR STATISTICS — FIT ONLY
# ============================================================

regime_sensor_stats_rows_628gb = []


for regime_628gb, group_628gb in (
    FD002_FIT_CORRUPTION_REFERENCE_628GB
    .groupby(
        "regime_id",
        sort=True
    )
):

    n_628gb = len(
        group_628gb
    )


    for sensor_628gb in (
        FD002_MODEL_SENSORS_628GB
    ):

        values_628gb = (
            group_628gb[
                sensor_628gb
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        mean_628gb = float(
            np.mean(
                values_628gb
            )
        )


        std_628gb = float(
            np.std(
                values_628gb,
                ddof=1
            )
        )


        if not np.isfinite(
            mean_628gb
        ):

            raise RuntimeError(
                f"Non-finite mean for "
                f"{sensor_628gb}, regime "
                f"{regime_628gb}."
            )


        if (
            not np.isfinite(
                std_628gb
            )
            or
            std_628gb <= 0.0
        ):

            raise RuntimeError(
                f"Invalid STD for "
                f"{sensor_628gb}, regime "
                f"{regime_628gb}: "
                f"{std_628gb}"
            )


        regime_sensor_stats_rows_628gb.append(
            {
                "regime_id":
                    int(
                        regime_628gb
                    ),

                "sensor":
                    sensor_628gb,

                "rows":
                    int(
                        n_628gb
                    ),

                "mean":
                    mean_628gb,

                "std":
                    std_628gb,
            }
        )


FD002_REGIME_SENSOR_STATS_628GB = (
    pd.DataFrame(
        regime_sensor_stats_rows_628gb
    )
)


expected_regime_sensor_rows_628gb = (
    6
    *
    15
)


if len(
    FD002_REGIME_SENSOR_STATS_628GB
) != expected_regime_sensor_rows_628gb:

    raise RuntimeError(
        "Unexpected regime × sensor "
        "reference size."
    )


print(
    "\nRegime × sensor statistics:",
    FD002_REGIME_SENSOR_STATS_628GB.shape
)


# ============================================================
# 8. POOLED WITHIN-REGIME SENSOR STANDARD DEVIATION
#
# For sensor s:
#
#   variance =
#       sum_r ((n_r - 1) * sigma_r^2)
#       --------------------------------
#       sum_r (n_r - 1)
#
# This removes between-operating-condition variance.
# ============================================================

pooled_scale_rows_628gb = []


for sensor_628gb in (
    FD002_MODEL_SENSORS_628GB
):

    stats_628gb = (
        FD002_REGIME_SENSOR_STATS_628GB[
            FD002_REGIME_SENSOR_STATS_628GB[
                "sensor"
            ]
            ==
            sensor_628gb
        ]
        .copy()
    )


    numerator_628gb = float(
        np.sum(
            (
                stats_628gb[
                    "rows"
                ].to_numpy(
                    dtype=np.float64
                )
                -
                1.0
            )
            *
            np.square(
                stats_628gb[
                    "std"
                ].to_numpy(
                    dtype=np.float64
                )
            )
        )
    )


    denominator_628gb = float(
        np.sum(
            stats_628gb[
                "rows"
            ].to_numpy(
                dtype=np.float64
            )
            -
            1.0
        )
    )


    pooled_std_628gb = float(
        np.sqrt(
            numerator_628gb
            /
            denominator_628gb
        )
    )


    global_std_628gb = float(
        np.std(
            FD002_FIT_CORRUPTION_REFERENCE_628GB[
                sensor_628gb
            ].to_numpy(
                dtype=np.float64
            ),
            ddof=1
        )
    )


    if (
        not np.isfinite(
            pooled_std_628gb
        )
        or
        pooled_std_628gb <= 0.0
    ):

        raise RuntimeError(
            f"Invalid pooled STD for "
            f"{sensor_628gb}."
        )


    pooled_scale_rows_628gb.append(
        {
            "sensor":
                sensor_628gb,

            "pooled_within_regime_std":
                pooled_std_628gb,

            "global_raw_std":
                global_std_628gb,

            "global_to_within_ratio":
                (
                    global_std_628gb
                    /
                    pooled_std_628gb
                ),
        }
    )


FD002_CORRUPTION_SCALE_REFERENCE_628GB = (
    pd.DataFrame(
        pooled_scale_rows_628gb
    )
    .set_index(
        "sensor"
    )
    .loc[
        FD002_MODEL_SENSORS_628GB
    ]
    .reset_index()
)


print(
    "\n"
    "--- CONDITION-ADJUSTED FD002 "
    "CORRUPTION SCALE ---"
)

display(
    FD002_CORRUPTION_SCALE_REFERENCE_628GB
    .round(
        6
    )
)


# ============================================================
# 9. REGIME-SPECIFIC IMPUTATION REFERENCE
#
# Used later for dropout only.
# Each missing value will be replaced using the FIT mean for:
#
#     current operating regime × affected sensor
#
# This avoids contaminating dropout with operating-condition shift.
# ============================================================

FD002_DROPOUT_IMPUTATION_REFERENCE_628GB = (
    FD002_REGIME_SENSOR_STATS_628GB[
        [
            "regime_id",
            "sensor",
            "mean",
        ]
    ]
    .copy()
)


print(
    "\n"
    "--- DROPOUT IMPUTATION REFERENCE ---"
)

print(
    "Rows:",
    len(
        FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
    )
)

print(
    "Expected:",
    6 * 15
)


# ============================================================
# 10. FREEZE TRANSFERRED FD002 CORRUPTION METHODOLOGY
# ============================================================

FD002_CORRUPTION_TRANSFER_CONTRACT_628GB = {

    "source_protocol":
        "frozen FD001 corruption protocol",

    "target_dataset":
        "NASA C-MAPSS FD002",

    "families": [
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ],

    "sensor_order":
        FD002_MODEL_SENSORS_628GB.copy(),

    "severity_mapping": {

        "bias":
            BIAS_SEVERITY_MAP_628GB.copy(),

        "noise":
            NOISE_SEVERITY_MAP_628GB.copy(),

        "drift":
            DRIFT_SEVERITY_MAP_628GB.copy(),

        "dropout":
            DROPOUT_SEVERITY_MAP_628GB.copy(),

        "stuck":
            STUCK_SEVERITY_MAP_628GB.copy(),
    },

    "bias_directions": [
        -1,
        1,
    ],

    "drift_directions": [
        -1,
        1,
    ],

    "noise_seed":
        42,

    "bias_scale_basis":
        (
            "FD002 FIT-only pooled "
            "within-operating-regime "
            "sensor standard deviation"
        ),

    "noise_scale_basis":
        (
            "FD002 FIT-only pooled "
            "within-operating-regime "
            "sensor standard deviation"
        ),

    "drift_scale_basis":
        (
            "FD002 FIT-only pooled "
            "within-operating-regime "
            "sensor standard deviation"
        ),

    "dropout_imputation":
        (
            "FD002 FIT-only "
            "regime-specific sensor mean"
        ),

    "corruption_space":
        "raw sensor measurement space",

    "corruption_stage":
        (
            "before condition-aware "
            "normalization and sequence creation"
        ),

    "official_test_labels_used":
        False,
}


# ============================================================
# 11. FINAL CONTRACT DISPLAY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "FD002 TRANSFERRED CORRUPTION CONTRACT\n"
    "============================================================"
)

print(
    "Bias severity map    :",
    BIAS_SEVERITY_MAP_628GB
)

print(
    "Noise severity map   :",
    NOISE_SEVERITY_MAP_628GB
)

print(
    "Drift severity map   :",
    DRIFT_SEVERITY_MAP_628GB
)

print(
    "Dropout severity map :",
    DROPOUT_SEVERITY_MAP_628GB
)

print(
    "Stuck severity map   :",
    STUCK_SEVERITY_MAP_628GB
)

print(
    "Bias directions      : [-1, +1]"
)

print(
    "Drift directions     : [-1, +1]"
)

print(
    "Noise seed           : 42"
)

print(
    "Magnitude reference  : "
    "FIT pooled within-regime STD"
)

print(
    "Dropout imputation   : "
    "FIT regime-specific mean"
)

print(
    "Corruption space     : raw sensor measurements"
)

print(
    "Model sensor order   :",
    FD002_MODEL_SENSORS_628GB
)


# ============================================================
# 12. FINAL INTEGRITY SUMMARY
# ============================================================

all_scale_values_finite_628gb = bool(
    np.isfinite(
        FD002_CORRUPTION_SCALE_REFERENCE_628GB[
            [
                "pooled_within_regime_std",
                "global_raw_std",
                "global_to_within_ratio",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 severity mapping exact : True"
)

print(
    "FD001/FD002 sensor sets match:",
    sensor_sets_match_628gb
)

print(
    "FD002 sensor count           :",
    len(
        FD002_MODEL_SENSORS_628GB
    )
)

print(
    "FD002 operating regimes      :",
    n_regimes_628gb
)

print(
    "Regime × sensor references   :",
    len(
        FD002_REGIME_SENSOR_STATS_628GB
    )
)

print(
    "Condition-adjusted scales    :",
    len(
        FD002_CORRUPTION_SCALE_REFERENCE_628GB
    )
)

print(
    "All scale values finite      :",
    all_scale_values_finite_628gb
)

print(
    "Scale fitted on              : FD002 FIT only"
)

print(
    "DEV used for scale fit       : False"
)

print(
    "Official TEST labels read    : False"
)

print(
    "TEST scored                  : False"
)

print(
    "Corruption applied           : False"
)

print(
    "Model retrained              : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28G-B ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-B-R1
# EXACT FD001 CORRUPTION-IMPLEMENTATION RECOVERY
#
# PURPOSE
# -------
# Recover the actual FD001 implementation code for:
#   bias, noise, drift, dropout, stuck
#
# Especially verify:
#   - dropout mask generation
#   - stuck-value definition
#   - drift ramp definition
#   - noise RNG behavior
#
# READ-ONLY SOURCE INSPECTION.
# Nothing is executed from the old notebook.
# ============================================================

from pathlib import Path
import json
import re


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-B-R1 — EXACT FD001 CORRUPTION SEMANTICS\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Locate project / source notebook
# ------------------------------------------------------------

cwd_628gbr1 = Path.cwd()

if (cwd_628gbr1 / "03_notebooks").exists():
    PROJECT_ROOT_628GBR1 = cwd_628gbr1

elif (cwd_628gbr1.parent / "03_notebooks").exists():
    PROJECT_ROOT_628GBR1 = cwd_628gbr1.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


SOURCE_NOTEBOOK_628GBR1 = (
    PROJECT_ROOT_628GBR1
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)


print("\n--- SOURCE CONTRACT ---")

print(
    "Project root :",
    PROJECT_ROOT_628GBR1
)

print(
    "Notebook     :",
    SOURCE_NOTEBOOK_628GBR1
)

print(
    "Exists       :",
    SOURCE_NOTEBOOK_628GBR1.exists()
)


if not SOURCE_NOTEBOOK_628GBR1.exists():
    raise FileNotFoundError(
        SOURCE_NOTEBOOK_628GBR1
    )


# ------------------------------------------------------------
# 2. Load notebook JSON only
# ------------------------------------------------------------

with open(
    SOURCE_NOTEBOOK_628GBR1,
    "r",
    encoding="utf-8"
) as f_628gbr1:

    nb_628gbr1 = json.load(
        f_628gbr1
    )


cells_628gbr1 = nb_628gbr1.get(
    "cells",
    []
)


# ------------------------------------------------------------
# 3. High-specificity implementation patterns
# ------------------------------------------------------------

patterns_628gbr1 = {

    "bias": [
        r"\bbias\b",
        r"alpha",
        r"direction",
    ],

    "noise": [
        r"\bnoise\b",
        r"normal\(",
        r"default_rng",
    ],

    "drift": [
        r"\bdrift\b",
        r"linspace",
        r"alpha",
    ],

    "dropout": [
        r"\bdropout\b",
        r"requested_fraction",
        r"random\(",
        r"choice\(",
        r"mask",
        r"imput",
    ],

    "stuck": [
        r"\bstuck\b",
        r"requested_fraction",
        r"ceil\(",
        r"floor\(",
        r"round\(",
    ],
}


# ------------------------------------------------------------
# 4. Score code cells
# ------------------------------------------------------------

candidate_cells_628gbr1 = []


for cell_index_628gbr1, cell_628gbr1 in enumerate(
    cells_628gbr1
):

    if cell_628gbr1.get(
        "cell_type"
    ) != "code":
        continue


    source_628gbr1 = "".join(
        cell_628gbr1.get(
            "source",
            []
        )
    )


    if not source_628gbr1.strip():
        continue


    family_hits_628gbr1 = {}


    for family_628gbr1, pats_628gbr1 in (
        patterns_628gbr1.items()
    ):

        hits_628gbr1 = sum(
            bool(
                re.search(
                    pat_628gbr1,
                    source_628gbr1,
                    flags=re.IGNORECASE
                )
            )
            for pat_628gbr1
            in pats_628gbr1
        )

        if hits_628gbr1 > 0:

            family_hits_628gbr1[
                family_628gbr1
            ] = hits_628gbr1


    # Keep only strongly relevant implementation cells.
    strong_score_628gbr1 = sum(
        family_hits_628gbr1.values()
    )


    if strong_score_628gbr1 >= 2:

        candidate_cells_628gbr1.append(
            {
                "cell_index":
                    cell_index_628gbr1,

                "score":
                    strong_score_628gbr1,

                "family_hits":
                    family_hits_628gbr1,

                "source":
                    source_628gbr1,
            }
        )


candidate_cells_628gbr1 = sorted(
    candidate_cells_628gbr1,
    key=lambda x_628gbr1: (
        -x_628gbr1["score"],
        x_628gbr1["cell_index"],
    )
)


# ------------------------------------------------------------
# 5. Print only the highest-value source cells
# ------------------------------------------------------------

print(
    "\n--- IMPLEMENTATION CELLS FOUND ---"
)

print(
    "Candidate cells:",
    len(
        candidate_cells_628gbr1
    )
)


MAX_PRINT_628GBR1 = 12


for rank_628gbr1, item_628gbr1 in enumerate(
    candidate_cells_628gbr1[
        :MAX_PRINT_628GBR1
    ],
    start=1
):

    print(
        "\n"
        "============================================================"
    )

    print(
        f"MATCH {rank_628gbr1:02d}"
    )

    print(
        "Cell index :",
        item_628gbr1[
            "cell_index"
        ]
    )

    print(
        "Score      :",
        item_628gbr1[
            "score"
        ]
    )

    print(
        "Family hits:",
        item_628gbr1[
            "family_hits"
        ]
    )

    print(
        "------------------------------------------------------------"
    )

    print(
        item_628gbr1[
            "source"
        ]
    )


# ------------------------------------------------------------
# 6. Extract highly relevant implementation lines
# ------------------------------------------------------------

important_line_pattern_628gbr1 = re.compile(
    r"("
    r"def\s+.*(?:bias|noise|drift|dropout|stuck)"
    r"|rng"
    r"|default_rng"
    r"|normal\("
    r"|linspace"
    r"|choice\("
    r"|random\("
    r"|requested_fraction"
    r"|actual_fraction"
    r"|stuck"
    r"|dropout"
    r"|imput"
    r"|direction"
    r"|alpha"
    r")",
    flags=re.IGNORECASE
)


important_lines_628gbr1 = []


for item_628gbr1 in candidate_cells_628gbr1:

    source_lines_628gbr1 = (
        item_628gbr1[
            "source"
        ]
        .splitlines()
    )


    for line_number_628gbr1, line_628gbr1 in enumerate(
        source_lines_628gbr1,
        start=1
    ):

        if important_line_pattern_628gbr1.search(
            line_628gbr1
        ):

            important_lines_628gbr1.append(
                {
                    "cell":
                        item_628gbr1[
                            "cell_index"
                        ],

                    "line":
                        line_number_628gbr1,

                    "text":
                        line_628gbr1.strip(),
                }
            )


print(
    "\n"
    "============================================================\n"
    "COMPACT IMPLEMENTATION-LINE REVIEW\n"
    "============================================================"
)


for row_628gbr1 in important_lines_628gbr1[
    :120
]:

    print(
        f"cell={row_628gbr1['cell']:03d} "
        f"| line={row_628gbr1['line']:03d} "
        f"| {row_628gbr1['text']}"
    )


# ------------------------------------------------------------
# 7. Audit object
# ------------------------------------------------------------

FD001_CORRUPTION_SEMANTICS_AUDIT_628GBR1 = {

    "source_notebook":
        str(
            SOURCE_NOTEBOOK_628GBR1
        ),

    "candidate_cells":
        int(
            len(
                candidate_cells_628gbr1
            )
        ),

    "implementation_lines":
        int(
            len(
                important_lines_628gbr1
            )
        ),

    "old_cells_executed":
        False,

    "corruption_applied":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-B-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Candidate implementation cells:",
    len(
        candidate_cells_628gbr1
    )
)

print(
    "Relevant implementation lines :",
    len(
        important_lines_628gbr1
    )
)

print(
    "Old notebook cells executed    : False"
)

print(
    "Corruption applied             : False"
)

print(
    "Official TEST labels read      : False"
)

print(
    "TEST scored                    : False"
)

print(
    "Model retrained                : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.28G-B-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-C
# FD002 CORRUPTION OPERATORS + SMOKE VALIDATION
#
# PURPOSE
# -------
# Implement the five frozen corruption families:
#
#   1. bias
#   2. noise
#   3. drift
#   4. dropout
#   5. stuck sensor
#
# Transfer rules:
#
#   - numerical severity coefficients come from frozen FD001
#   - bias/noise/drift magnitude uses FD002 FIT-only
#     pooled within-operating-regime sensor STD
#   - dropout uses exact-count seeded permutation masks
#   - dropout imputation uses FD002 FIT-only
#     regime-specific sensor means
#   - stuck sensor freezes the final fraction of EACH engine
#     at the last valid value before the stuck region
#
# THIS STEP IS ONLY A SMOKE / CONTRACT VALIDATION.
#
# NO official TEST labels.
# NO TEST scoring.
# NO threshold selection.
# NO model retraining.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-C — FD002 CORRUPTION OPERATORS + SMOKE TEST\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628gc = [

    "FD002_DEV_TARGETED_628B",
    "FD002_DEV_NORMALIZED_628C",

    "FD002_MODEL_SENSORS_628GB",

    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",

    "BIAS_SEVERITY_MAP_628GB",
    "NOISE_SEVERITY_MAP_628GB",
    "DRIFT_SEVERITY_MAP_628GB",
    "DROPOUT_SEVERITY_MAP_628GB",
    "STUCK_SEVERITY_MAP_628GB",

    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
]


print(
    "\n--- REQUIRED OBJECTS ---"
)


missing_628gc = []


for name_628gc in required_628gc:

    available_628gc = (
        name_628gc in globals()
    )

    print(
        f"{name_628gc:<48}: "
        f"{available_628gc}"
    )

    if not available_628gc:

        missing_628gc.append(
            name_628gc
        )


if missing_628gc:

    raise RuntimeError(
        "Missing required objects: "
        + str(
            missing_628gc
        )
    )


# ============================================================
# 2. ALIGN RAW DEV WITH FROZEN OPERATING-REGIME IDS
# ============================================================

dev_raw_628gc = (
    FD002_DEV_TARGETED_628B
    .copy()
    .reset_index(
        drop=True
    )
)


dev_norm_628gc = (
    FD002_DEV_NORMALIZED_628C
    .copy()
    .reset_index(
        drop=True
    )
)


key_cols_628gc = [
    "unit_id",
    "cycle",
]


for col_628gc in key_cols_628gc:

    if col_628gc not in dev_raw_628gc.columns:

        raise RuntimeError(
            f"{col_628gc} missing from raw DEV."
        )

    if col_628gc not in dev_norm_628gc.columns:

        raise RuntimeError(
            f"{col_628gc} missing from normalized DEV."
        )


if "regime_id" not in dev_norm_628gc.columns:

    raise RuntimeError(
        "regime_id missing from normalized DEV."
    )


exact_dev_alignment_628gc = bool(

    dev_raw_628gc[
        key_cols_628gc
    ].equals(

        dev_norm_628gc[
            key_cols_628gc
        ]
    )
)


print(
    "\n--- DEV ALIGNMENT ---"
)

print(
    "RAW DEV rows       :",
    len(
        dev_raw_628gc
    )
)

print(
    "Normalized DEV rows:",
    len(
        dev_norm_628gc
    )
)

print(
    "Exact key alignment:",
    exact_dev_alignment_628gc
)


if not exact_dev_alignment_628gc:

    raise RuntimeError(
        "RAW and normalized DEV rows "
        "are not exactly aligned."
    )


FD002_DEV_CORRUPTION_BASE_628GC = (
    dev_raw_628gc.copy()
)


FD002_DEV_CORRUPTION_BASE_628GC[
    "regime_id"
] = (
    dev_norm_628gc[
        "regime_id"
    ]
    .to_numpy()
)


# ============================================================
# 3. BUILD FIT-ONLY SCALE LOOKUP
# ============================================================

scale_table_628gc = (
    FD002_CORRUPTION_SCALE_REFERENCE_628GB
    .copy()
)


required_scale_cols_628gc = [
    "sensor",
    "pooled_within_regime_std",
]


for col_628gc in required_scale_cols_628gc:

    if col_628gc not in scale_table_628gc.columns:

        raise RuntimeError(
            f"{col_628gc} missing from "
            "corruption scale table."
        )


SENSOR_SCALE_628GC = {

    str(
        row_628gc["sensor"]
    ):
        float(
            row_628gc[
                "pooled_within_regime_std"
            ]
        )

    for _, row_628gc
    in scale_table_628gc.iterrows()
}


for sensor_628gc in (
    FD002_MODEL_SENSORS_628GB
):

    if sensor_628gc not in SENSOR_SCALE_628GC:

        raise RuntimeError(
            f"No FIT scale for {sensor_628gc}."
        )


    value_628gc = SENSOR_SCALE_628GC[
        sensor_628gc
    ]


    if (
        not np.isfinite(
            value_628gc
        )
        or
        value_628gc <= 0.0
    ):

        raise RuntimeError(
            f"Invalid FIT scale for "
            f"{sensor_628gc}: "
            f"{value_628gc}"
        )


# ============================================================
# 4. BUILD FIT-ONLY REGIME-SPECIFIC DROPOUT MEAN LOOKUP
# ============================================================

imputation_table_628gc = (
    FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
    .copy()
)


required_impute_cols_628gc = [
    "regime_id",
    "sensor",
    "mean",
]


for col_628gc in required_impute_cols_628gc:

    if col_628gc not in imputation_table_628gc.columns:

        raise RuntimeError(
            f"{col_628gc} missing from "
            "dropout imputation reference."
        )


DROPOUT_MEAN_628GC = {

    (
        int(
            row_628gc["regime_id"]
        ),
        str(
            row_628gc["sensor"]
        )
    ):
        float(
            row_628gc["mean"]
        )

    for _, row_628gc
    in imputation_table_628gc.iterrows()
}


# ============================================================
# 5. COMMON VALIDATION HELPER
# ============================================================

def validate_sensor_628gc(
    df,
    sensor
):

    if sensor not in (
        FD002_MODEL_SENSORS_628GB
    ):

        raise ValueError(
            f"{sensor} is not one of the "
            "15 frozen FD002 model sensors."
        )


    if sensor not in df.columns:

        raise ValueError(
            f"{sensor} is missing from dataframe."
        )


# ============================================================
# 6. BIAS OPERATOR
#
# frozen semantics:
#
#     corrupted =
#         clean
#         + direction * alpha * FIT_scale(sensor)
# ============================================================

def apply_fd002_bias_628gc(
    df,
    sensor,
    alpha,
    direction
):

    validate_sensor_628gc(
        df,
        sensor
    )


    if direction not in (
        -1,
        1
    ):

        raise ValueError(
            "direction must be -1 or +1."
        )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    scale = SENSOR_SCALE_628GC[
        sensor
    ]


    bias_amount = float(
        direction
        *
        alpha
        *
        scale
    )


    corrupted = (
        df.copy()
    )


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        bias_amount
    )


    return (
        corrupted,
        bias_amount
    )


# ============================================================
# 7. NOISE OPERATOR
#
# frozen semantics:
#
# zero-mean Gaussian additive noise
#
# The same seed generates the same standard-normal realization
# for low/medium/high severity; severity only rescales it.
#
# This preserves an ordered severity experiment.
# ============================================================

def apply_fd002_noise_628gc(
    df,
    sensor,
    alpha,
    seed=42
):

    validate_sensor_628gc(
        df,
        sensor
    )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    scale = SENSOR_SCALE_628GC[
        sensor
    ]


    target_std = float(
        alpha
        *
        scale
    )


    rng = np.random.default_rng(
        int(
            seed
        )
    )


    base_noise = rng.normal(
        loc=0.0,
        scale=1.0,
        size=len(
            df
        )
    )


    noise = (
        base_noise
        *
        target_std
    )


    corrupted = (
        df.copy()
    )


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        noise
    )


    return (
        corrupted,
        noise,
        target_std
    )


# ============================================================
# 8. DRIFT OPERATOR
#
# Per engine:
#
# first observed cycle -> drift = 0
# last observed cycle  -> drift =
#      direction * alpha * FIT_scale(sensor)
#
# Linear interpolation occurs independently within each engine.
# ============================================================

def apply_fd002_drift_628gc(
    df,
    sensor,
    alpha,
    direction
):

    validate_sensor_628gc(
        df,
        sensor
    )


    if direction not in (
        -1,
        1
    ):

        raise ValueError(
            "direction must be -1 or +1."
        )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    final_drift = float(
        direction
        *
        alpha
        *
        SENSOR_SCALE_628GC[
            sensor
        ]
    )


    corrupted = (
        df.copy()
    )


    drift_values = np.zeros(
        len(
            corrupted
        ),
        dtype=np.float64
    )


    for _, group in (
        corrupted
        .groupby(
            "unit_id",
            sort=False
        )
    ):

        group_sorted = (
            group
            .sort_values(
                "cycle"
            )
        )


        positions = (
            group_sorted.index
            .to_numpy(
                dtype=np.int64
            )
        )


        n_rows = len(
            positions
        )


        if n_rows == 1:

            engine_drift = np.array(
                [
                    final_drift
                ],
                dtype=np.float64
            )

        else:

            engine_drift = np.linspace(
                0.0,
                final_drift,
                num=n_rows,
                endpoint=True,
                dtype=np.float64
            )


        drift_values[
            positions
        ] = engine_drift


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        drift_values
    )


    return (
        corrupted,
        drift_values,
        final_drift
    )


# ============================================================
# 9. DROPOUT OPERATOR
#
# Frozen FD001 semantics:
#
#   n_drop = round(rate * N)
#
#   rng = default_rng(seed)
#
#   random_order = rng.permutation(N)
#
#   drop_positions = random_order[:n_drop]
#
# Therefore severity masks are nested when the same seed is used.
#
# FD002 adaptation:
# missing values are imputed with:
#
#   FIT-only regime-specific sensor mean
# ============================================================

def apply_fd002_dropout_628gc(
    df,
    sensor,
    dropout_rate,
    seed=42
):

    validate_sensor_628gc(
        df,
        sensor
    )


    dropout_rate = float(
        dropout_rate
    )


    if not (
        0.0
        <=
        dropout_rate
        <
        1.0
    ):

        raise ValueError(
            "dropout_rate must satisfy "
            "0 <= rate < 1."
        )


    if "regime_id" not in df.columns:

        raise ValueError(
            "regime_id is required for "
            "condition-aware imputation."
        )


    corrupted = (
        df.copy()
    )


    n_rows = len(
        corrupted
    )


    n_drop = int(
        round(
            dropout_rate
            *
            n_rows
        )
    )


    rng = np.random.default_rng(
        int(
            seed
        )
    )


    random_order = rng.permutation(
        n_rows
    )


    drop_positions = random_order[
        :n_drop
    ]


    drop_mask = np.zeros(
        n_rows,
        dtype=bool
    )


    drop_mask[
        drop_positions
    ] = True


    sensor_col_pos = (
        corrupted.columns
        .get_loc(
            sensor
        )
    )


    corrupted.iloc[
        drop_positions,
        sensor_col_pos
    ] = np.nan


    # --------------------------------------------------------
    # FIT-only regime-specific imputation
    # --------------------------------------------------------

    regime_values = (
        corrupted[
            "regime_id"
        ]
        .to_numpy()
    )


    imputation_values = np.empty(
        n_rows,
        dtype=np.float64
    )


    for position_628gc in range(
        n_rows
    ):

        regime_628gc = int(
            regime_values[
                position_628gc
            ]
        )


        key_628gc = (
            regime_628gc,
            sensor
        )


        if key_628gc not in DROPOUT_MEAN_628GC:

            raise RuntimeError(
                "Missing FIT dropout mean for "
                f"regime={regime_628gc}, "
                f"sensor={sensor}."
            )


        imputation_values[
            position_628gc
        ] = (
            DROPOUT_MEAN_628GC[
                key_628gc
            ]
        )


    missing_positions = np.flatnonzero(
        drop_mask
    )


    corrupted.iloc[
        missing_positions,
        sensor_col_pos
    ] = (
        imputation_values[
            missing_positions
        ]
    )


    return (
        corrupted,
        drop_mask,
        imputation_values
    )


# ============================================================
# 10. STUCK-SENSOR OPERATOR
#
# Frozen FD001 semantics:
#
# For EACH engine:
#
#   n_affected =
#       round(fraction * trajectory_length)
#
# constrained to:
#
#   at least 1 affected point
#   at least 1 valid point before onset
#
# affected suffix is frozen to the last valid reading before onset.
# ============================================================

def apply_fd002_stuck_628gc(
    df,
    sensor,
    affected_fraction
):

    validate_sensor_628gc(
        df,
        sensor
    )


    affected_fraction = float(
        affected_fraction
    )


    if not (
        0.0
        <
        affected_fraction
        <
        1.0
    ):

        raise ValueError(
            "affected_fraction must satisfy "
            "0 < fraction < 1."
        )


    corrupted = (
        df.copy()
    )


    affected_mask = np.zeros(
        len(
            corrupted
        ),
        dtype=bool
    )


    records_628gc = []


    for unit_id_628gc, group_628gc in (
        corrupted
        .groupby(
            "unit_id",
            sort=False
        )
    ):

        group_sorted_628gc = (
            group_628gc
            .sort_values(
                "cycle"
            )
        )


        positions_628gc = (
            group_sorted_628gc.index
            .to_numpy(
                dtype=np.int64
            )
        )


        n_rows_628gc = len(
            positions_628gc
        )


        if n_rows_628gc < 2:

            raise RuntimeError(
                "Stuck corruption requires "
                "at least two cycles per engine."
            )


        n_affected_628gc = int(
            round(
                affected_fraction
                *
                n_rows_628gc
            )
        )


        n_affected_628gc = max(
            1,
            n_affected_628gc
        )


        n_affected_628gc = min(
            n_affected_628gc,
            n_rows_628gc - 1
        )


        onset_position_628gc = (
            n_rows_628gc
            -
            n_affected_628gc
        )


        source_position_628gc = (
            positions_628gc[
                onset_position_628gc
                -
                1
            ]
        )


        affected_positions_628gc = (
            positions_628gc[
                onset_position_628gc:
            ]
        )


        freeze_value_628gc = float(
            corrupted.loc[
                source_position_628gc,
                sensor
            ]
        )


        corrupted.loc[
            affected_positions_628gc,
            sensor
        ] = freeze_value_628gc


        affected_mask[
            affected_positions_628gc
        ] = True


        records_628gc.append(
            {
                "unit_id":
                    unit_id_628gc,

                "trajectory_length":
                    n_rows_628gc,

                "requested_fraction":
                    affected_fraction,

                "affected_count":
                    n_affected_628gc,

                "actual_fraction":
                    (
                        n_affected_628gc
                        /
                        n_rows_628gc
                    ),

                "source_cycle":
                    float(
                        group_sorted_628gc
                        .iloc[
                            onset_position_628gc
                            -
                            1
                        ][
                            "cycle"
                        ]
                    ),

                "stuck_start_cycle":
                    float(
                        group_sorted_628gc
                        .iloc[
                            onset_position_628gc
                        ][
                            "cycle"
                        ]
                    ),

                "freeze_value":
                    freeze_value_628gc,
            }
        )


    stuck_info_628gc = pd.DataFrame(
        records_628gc
    )


    return (
        corrupted,
        affected_mask,
        stuck_info_628gc
    )


# ============================================================
# 11. SMOKE-TEST SENSOR
# ============================================================

SMOKE_SENSOR_628GC = (
    FD002_MODEL_SENSORS_628GB[
        0
    ]
)


print(
    "\n--- SMOKE-TEST SENSOR ---"
)

print(
    "Sensor:",
    SMOKE_SENSOR_628GC
)

print(
    "FIT pooled within-regime STD:",
    SENSOR_SCALE_628GC[
        SMOKE_SENSOR_628GC
    ]
)


# Preserve clean DEV values for mutation audit.
clean_dev_sensor_snapshot_628gc = (
    FD002_DEV_CORRUPTION_BASE_628GC[
        FD002_MODEL_SENSORS_628GB
    ]
    .to_numpy(
        dtype=np.float64,
        copy=True
    )
)


# ============================================================
# 12. BIAS SMOKE TEST
# ============================================================

bias_alpha_628gc = (
    BIAS_SEVERITY_MAP_628GB[
        "medium"
    ]
)


bias_pos_df_628gc, bias_pos_amount_628gc = (
    apply_fd002_bias_628gc(
        FD002_DEV_CORRUPTION_BASE_628GC,
        SMOKE_SENSOR_628GC,
        bias_alpha_628gc,
        +1
    )
)


bias_neg_df_628gc, bias_neg_amount_628gc = (
    apply_fd002_bias_628gc(
        FD002_DEV_CORRUPTION_BASE_628GC,
        SMOKE_SENSOR_628GC,
        bias_alpha_628gc,
        -1
    )
)


bias_pos_shift_628gc = (
    bias_pos_df_628gc[
        SMOKE_SENSOR_628GC
    ].to_numpy(
        dtype=np.float64
    )
    -
    FD002_DEV_CORRUPTION_BASE_628GC[
        SMOKE_SENSOR_628GC
    ].to_numpy(
        dtype=np.float64
    )
)


bias_neg_shift_628gc = (
    bias_neg_df_628gc[
        SMOKE_SENSOR_628GC
    ].to_numpy(
        dtype=np.float64
    )
    -
    FD002_DEV_CORRUPTION_BASE_628GC[
        SMOKE_SENSOR_628GC
    ].to_numpy(
        dtype=np.float64
    )
)


bias_positive_exact_628gc = bool(
    np.allclose(
        bias_pos_shift_628gc,
        bias_pos_amount_628gc,
        atol=1e-12,
        rtol=1e-12
    )
)


bias_negative_exact_628gc = bool(
    np.allclose(
        bias_neg_shift_628gc,
        bias_neg_amount_628gc,
        atol=1e-12,
        rtol=1e-12
    )
)


bias_symmetry_628gc = bool(
    np.isclose(
        bias_pos_amount_628gc,
        -bias_neg_amount_628gc,
        atol=1e-12,
        rtol=1e-12
    )
)


# ============================================================
# 13. NOISE SMOKE TEST
# ============================================================

noise_results_628gc = {}


for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    alpha_628gc = (
        NOISE_SEVERITY_MAP_628GB[
            severity_628gc
        ]
    )


    (
        noise_df_628gc,
        noise_vector_628gc,
        target_noise_std_628gc
    ) = apply_fd002_noise_628gc(

        FD002_DEV_CORRUPTION_BASE_628GC,

        SMOKE_SENSOR_628GC,

        alpha_628gc,

        seed=42
    )


    noise_results_628gc[
        severity_628gc
    ] = {

        "alpha":
            alpha_628gc,

        "df":
            noise_df_628gc,

        "noise":
            noise_vector_628gc,

        "target_std":
            target_noise_std_628gc,

        "actual_mean":
            float(
                np.mean(
                    noise_vector_628gc
                )
            ),

        "actual_std":
            float(
                np.std(
                    noise_vector_628gc,
                    ddof=0
                )
            ),
    }


noise_low_scaled_628gc = (
    noise_results_628gc[
        "low"
    ][
        "noise"
    ]
    /
    noise_results_628gc[
        "low"
    ][
        "alpha"
    ]
)


noise_medium_scaled_628gc = (
    noise_results_628gc[
        "medium"
    ][
        "noise"
    ]
    /
    noise_results_628gc[
        "medium"
    ][
        "alpha"
    ]
)


noise_high_scaled_628gc = (
    noise_results_628gc[
        "high"
    ][
        "noise"
    ]
    /
    noise_results_628gc[
        "high"
    ][
        "alpha"
    ]
)


noise_shared_realization_628gc = bool(

    np.allclose(
        noise_low_scaled_628gc,
        noise_medium_scaled_628gc,
        atol=1e-12,
        rtol=1e-12
    )

    and

    np.allclose(
        noise_low_scaled_628gc,
        noise_high_scaled_628gc,
        atol=1e-12,
        rtol=1e-12
    )
)


# ============================================================
# 14. DRIFT SMOKE TEST
# ============================================================

drift_alpha_628gc = (
    DRIFT_SEVERITY_MAP_628GB[
        "medium"
    ]
)


(
    drift_df_628gc,
    drift_values_628gc,
    drift_final_628gc
) = apply_fd002_drift_628gc(

    FD002_DEV_CORRUPTION_BASE_628GC,

    SMOKE_SENSOR_628GC,

    drift_alpha_628gc,

    +1
)


drift_audit_df_628gc = pd.DataFrame(
    {
        "unit_id":
            FD002_DEV_CORRUPTION_BASE_628GC[
                "unit_id"
            ].to_numpy(),

        "cycle":
            FD002_DEV_CORRUPTION_BASE_628GC[
                "cycle"
            ].to_numpy(),

        "drift":
            drift_values_628gc,
    }
)


drift_first_628gc = (
    drift_audit_df_628gc
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .groupby(
        "unit_id"
    )[
        "drift"
    ]
    .first()
)


drift_last_628gc = (
    drift_audit_df_628gc
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .groupby(
        "unit_id"
    )[
        "drift"
    ]
    .last()
)


drift_starts_zero_628gc = bool(
    np.allclose(
        drift_first_628gc.to_numpy(),
        0.0,
        atol=1e-12,
        rtol=1e-12
    )
)


drift_ends_exact_628gc = bool(
    np.allclose(
        drift_last_628gc.to_numpy(),
        drift_final_628gc,
        atol=1e-12,
        rtol=1e-12
    )
)


# ============================================================
# 15. DROPOUT SMOKE TEST
# ============================================================

dropout_results_628gc = {}


for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    fraction_628gc = (
        DROPOUT_SEVERITY_MAP_628GB[
            severity_628gc
        ]
    )


    (
        dropout_df_628gc,
        dropout_mask_628gc,
        dropout_impute_values_628gc
    ) = apply_fd002_dropout_628gc(

        FD002_DEV_CORRUPTION_BASE_628GC,

        SMOKE_SENSOR_628GC,

        fraction_628gc,

        seed=42
    )


    expected_count_628gc = int(
        round(
            fraction_628gc
            *
            len(
                FD002_DEV_CORRUPTION_BASE_628GC
            )
        )
    )


    actual_count_628gc = int(
        dropout_mask_628gc.sum()
    )


    original_values_628gc = (
        FD002_DEV_CORRUPTION_BASE_628GC[
            SMOKE_SENSOR_628GC
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    corrupted_values_628gc = (
        dropout_df_628gc[
            SMOKE_SENSOR_628GC
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    dropped_match_imputation_628gc = bool(
        np.allclose(
            corrupted_values_628gc[
                dropout_mask_628gc
            ],
            dropout_impute_values_628gc[
                dropout_mask_628gc
            ],
            atol=1e-12,
            rtol=1e-12
        )
    )


    unchanged_elsewhere_628gc = bool(
        np.allclose(
            corrupted_values_628gc[
                ~dropout_mask_628gc
            ],
            original_values_628gc[
                ~dropout_mask_628gc
            ],
            atol=0.0,
            rtol=0.0
        )
    )


    dropout_results_628gc[
        severity_628gc
    ] = {

        "fraction":
            fraction_628gc,

        "mask":
            dropout_mask_628gc,

        "expected_count":
            expected_count_628gc,

        "actual_count":
            actual_count_628gc,

        "count_exact":
            (
                expected_count_628gc
                ==
                actual_count_628gc
            ),

        "imputation_exact":
            dropped_match_imputation_628gc,

        "clean_elsewhere":
            unchanged_elsewhere_628gc,
    }


dropout_nested_628gc = bool(

    np.all(
        dropout_results_628gc[
            "low"
        ][
            "mask"
        ]
        <=
        dropout_results_628gc[
            "medium"
        ][
            "mask"
        ]
    )

    and

    np.all(
        dropout_results_628gc[
            "medium"
        ][
            "mask"
        ]
        <=
        dropout_results_628gc[
            "high"
        ][
            "mask"
        ]
    )
)


# ============================================================
# 16. STUCK-SENSOR SMOKE TEST
# ============================================================

stuck_results_628gc = {}


for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    fraction_628gc = (
        STUCK_SEVERITY_MAP_628GB[
            severity_628gc
        ]
    )


    (
        stuck_df_628gc,
        stuck_mask_628gc,
        stuck_info_628gc
    ) = apply_fd002_stuck_628gc(

        FD002_DEV_CORRUPTION_BASE_628GC,

        SMOKE_SENSOR_628GC,

        fraction_628gc
    )


    original_628gc = (
        FD002_DEV_CORRUPTION_BASE_628GC
    )


    all_engines_exact_628gc = True


    for _, info_628gc in (
        stuck_info_628gc.iterrows()
    ):

        unit_id_628gc = (
            info_628gc[
                "unit_id"
            ]
        )


        n_affected_628gc = int(
            info_628gc[
                "affected_count"
            ]
        )


        original_engine_628gc = (
            original_628gc[
                original_628gc[
                    "unit_id"
                ]
                ==
                unit_id_628gc
            ]
            .sort_values(
                "cycle"
            )
        )


        corrupted_engine_628gc = (
            stuck_df_628gc[
                stuck_df_628gc[
                    "unit_id"
                ]
                ==
                unit_id_628gc
            ]
            .sort_values(
                "cycle"
            )
        )


        onset_628gc = (
            len(
                original_engine_628gc
            )
            -
            n_affected_628gc
        )


        freeze_value_628gc = float(
            original_engine_628gc[
                SMOKE_SENSOR_628GC
            ]
            .iloc[
                onset_628gc
                -
                1
            ]
        )


        before_ok_628gc = bool(
            np.allclose(
                corrupted_engine_628gc[
                    SMOKE_SENSOR_628GC
                ]
                .iloc[
                    :onset_628gc
                ]
                .to_numpy(
                    dtype=np.float64
                ),

                original_engine_628gc[
                    SMOKE_SENSOR_628GC
                ]
                .iloc[
                    :onset_628gc
                ]
                .to_numpy(
                    dtype=np.float64
                ),

                atol=0.0,
                rtol=0.0
            )
        )


        after_ok_628gc = bool(
            np.allclose(
                corrupted_engine_628gc[
                    SMOKE_SENSOR_628GC
                ]
                .iloc[
                    onset_628gc:
                ]
                .to_numpy(
                    dtype=np.float64
                ),

                freeze_value_628gc,

                atol=1e-12,
                rtol=1e-12
            )
        )


        if not (
            before_ok_628gc
            and
            after_ok_628gc
        ):

            all_engines_exact_628gc = False

            break


    stuck_results_628gc[
        severity_628gc
    ] = {

        "fraction":
            fraction_628gc,

        "mask":
            stuck_mask_628gc,

        "affected_rows":
            int(
                stuck_mask_628gc.sum()
            ),

        "engines":
            int(
                len(
                    stuck_info_628gc
                )
            ),

        "exact":
            all_engines_exact_628gc,
    }


stuck_nested_628gc = bool(

    np.all(
        stuck_results_628gc[
            "low"
        ][
            "mask"
        ]
        <=
        stuck_results_628gc[
            "medium"
        ][
            "mask"
        ]
    )

    and

    np.all(
        stuck_results_628gc[
            "medium"
        ][
            "mask"
        ]
        <=
        stuck_results_628gc[
            "high"
        ][
            "mask"
        ]
    )
)


# ============================================================
# 17. SOURCE-MUTATION AUDIT
# ============================================================

post_smoke_clean_values_628gc = (
    FD002_DEV_CORRUPTION_BASE_628GC[
        FD002_MODEL_SENSORS_628GB
    ]
    .to_numpy(
        dtype=np.float64
    )
)


source_unchanged_628gc = bool(
    np.array_equal(
        clean_dev_sensor_snapshot_628gc,
        post_smoke_clean_values_628gc
    )
)


# ============================================================
# 18. FINITE-VALUE AUDIT
# ============================================================

smoke_frames_628gc = [

    bias_pos_df_628gc,
    bias_neg_df_628gc,

    noise_results_628gc[
        "low"
    ][
        "df"
    ],

    noise_results_628gc[
        "medium"
    ][
        "df"
    ],

    noise_results_628gc[
        "high"
    ][
        "df"
    ],

    drift_df_628gc,
]


all_smoke_finite_628gc = True


for frame_628gc in smoke_frames_628gc:

    values_628gc = (
        frame_628gc[
            FD002_MODEL_SENSORS_628GB
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    if not np.isfinite(
        values_628gc
    ).all():

        all_smoke_finite_628gc = False


# Dropout and stuck should also be finite after imputation/freeze.
for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    dropout_frame_628gc, _, _ = (
        apply_fd002_dropout_628gc(

            FD002_DEV_CORRUPTION_BASE_628GC,

            SMOKE_SENSOR_628GC,

            DROPOUT_SEVERITY_MAP_628GB[
                severity_628gc
            ],

            seed=42
        )
    )


    stuck_frame_628gc, _, _ = (
        apply_fd002_stuck_628gc(

            FD002_DEV_CORRUPTION_BASE_628GC,

            SMOKE_SENSOR_628GC,

            STUCK_SEVERITY_MAP_628GB[
                severity_628gc
            ]
        )
    )


    if not np.isfinite(

        dropout_frame_628gc[
            FD002_MODEL_SENSORS_628GB
        ].to_numpy(
            dtype=np.float64
        )

    ).all():

        all_smoke_finite_628gc = False


    if not np.isfinite(

        stuck_frame_628gc[
            FD002_MODEL_SENSORS_628GB
        ].to_numpy(
            dtype=np.float64
        )

    ).all():

        all_smoke_finite_628gc = False


# ============================================================
# 19. COMPACT VALIDATION TABLE
# ============================================================

validation_rows_628gc = [

    {
        "check":
            "bias_positive_exact",
        "passed":
            bias_positive_exact_628gc,
    },

    {
        "check":
            "bias_negative_exact",
        "passed":
            bias_negative_exact_628gc,
    },

    {
        "check":
            "bias_direction_symmetry",
        "passed":
            bias_symmetry_628gc,
    },

    {
        "check":
            "noise_shared_seed_realization",
        "passed":
            noise_shared_realization_628gc,
    },

    {
        "check":
            "drift_starts_at_zero",
        "passed":
            drift_starts_zero_628gc,
    },

    {
        "check":
            "drift_ends_at_exact_magnitude",
        "passed":
            drift_ends_exact_628gc,
    },

    {
        "check":
            "dropout_low_count_exact",
        "passed":
            dropout_results_628gc[
                "low"
            ][
                "count_exact"
            ],
    },

    {
        "check":
            "dropout_medium_count_exact",
        "passed":
            dropout_results_628gc[
                "medium"
            ][
                "count_exact"
            ],
    },

    {
        "check":
            "dropout_high_count_exact",
        "passed":
            dropout_results_628gc[
                "high"
            ][
                "count_exact"
            ],
    },

    {
        "check":
            "dropout_nested_masks",
        "passed":
            dropout_nested_628gc,
    },

    {
        "check":
            "dropout_regime_imputation_exact",
        "passed":
            all(
                dropout_results_628gc[
                    sev_628gc
                ][
                    "imputation_exact"
                ]

                for sev_628gc
                in [
                    "low",
                    "medium",
                    "high",
                ]
            ),
    },

    {
        "check":
            "dropout_clean_elsewhere",
        "passed":
            all(
                dropout_results_628gc[
                    sev_628gc
                ][
                    "clean_elsewhere"
                ]

                for sev_628gc
                in [
                    "low",
                    "medium",
                    "high",
                ]
            ),
    },

    {
        "check":
            "stuck_all_engines_exact",
        "passed":
            all(
                stuck_results_628gc[
                    sev_628gc
                ][
                    "exact"
                ]

                for sev_628gc
                in [
                    "low",
                    "medium",
                    "high",
                ]
            ),
    },

    {
        "check":
            "stuck_nested_regions",
        "passed":
            stuck_nested_628gc,
    },

    {
        "check":
            "clean_source_unchanged",
        "passed":
            source_unchanged_628gc,
    },

    {
        "check":
            "all_smoke_outputs_finite",
        "passed":
            all_smoke_finite_628gc,
    },
]


FD002_CORRUPTION_SMOKE_AUDIT_628GC = (
    pd.DataFrame(
        validation_rows_628gc
    )
)


print(
    "\n--- CORRUPTION OPERATOR VALIDATION ---"
)

display(
    FD002_CORRUPTION_SMOKE_AUDIT_628GC
)


all_contract_checks_pass_628gc = bool(
    FD002_CORRUPTION_SMOKE_AUDIT_628GC[
        "passed"
    ].all()
)


# ============================================================
# 20. COMPACT FAMILY SUMMARY
# ============================================================

print(
    "\n--- BIAS SMOKE TEST ---"
)

print(
    "Medium alpha        :",
    bias_alpha_628gc
)

print(
    "Positive amount     :",
    bias_pos_amount_628gc
)

print(
    "Negative amount     :",
    bias_neg_amount_628gc
)

print(
    "Positive exact      :",
    bias_positive_exact_628gc
)

print(
    "Negative exact      :",
    bias_negative_exact_628gc
)


print(
    "\n--- NOISE SMOKE TEST ---"
)

for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    info_628gc = (
        noise_results_628gc[
            severity_628gc
        ]
    )

    print(
        f"{severity_628gc:<6} "
        f"alpha={info_628gc['alpha']:.4f} "
        f"target_std={info_628gc['target_std']:.6f} "
        f"actual_std={info_628gc['actual_std']:.6f} "
        f"mean={info_628gc['actual_mean']:.6f}"
    )


print(
    "Shared realization:",
    noise_shared_realization_628gc
)


print(
    "\n--- DRIFT SMOKE TEST ---"
)

print(
    "Medium alpha       :",
    drift_alpha_628gc
)

print(
    "Final magnitude    :",
    drift_final_628gc
)

print(
    "All starts at zero :",
    drift_starts_zero_628gc
)

print(
    "All ends exact     :",
    drift_ends_exact_628gc
)


print(
    "\n--- DROPOUT SMOKE TEST ---"
)

for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    info_628gc = (
        dropout_results_628gc[
            severity_628gc
        ]
    )

    print(
        f"{severity_628gc:<6} "
        f"fraction={info_628gc['fraction']:.2f} "
        f"expected={info_628gc['expected_count']} "
        f"actual={info_628gc['actual_count']} "
        f"impute_exact={info_628gc['imputation_exact']}"
    )


print(
    "Nested masks:",
    dropout_nested_628gc
)


print(
    "\n--- STUCK-SENSOR SMOKE TEST ---"
)

for severity_628gc in [
    "low",
    "medium",
    "high",
]:

    info_628gc = (
        stuck_results_628gc[
            severity_628gc
        ]
    )

    print(
        f"{severity_628gc:<6} "
        f"fraction={info_628gc['fraction']:.2f} "
        f"affected_rows={info_628gc['affected_rows']} "
        f"engines={info_628gc['engines']} "
        f"exact={info_628gc['exact']}"
    )


print(
    "Nested stuck regions:",
    stuck_nested_628gc
)


# ============================================================
# 21. FREEZE OPERATOR CONTRACT IN MEMORY
# ============================================================

FD002_CORRUPTION_OPERATOR_CONTRACT_628GC = {

    "bias":
        apply_fd002_bias_628gc,

    "noise":
        apply_fd002_noise_628gc,

    "drift":
        apply_fd002_drift_628gc,

    "dropout":
        apply_fd002_dropout_628gc,

    "stuck":
        apply_fd002_stuck_628gc,

    "sensor_scale":
        SENSOR_SCALE_628GC.copy(),

    "dropout_mean_reference":
        DROPOUT_MEAN_628GC.copy(),

    "noise_seed":
        42,

    "smoke_sensor":
        SMOKE_SENSOR_628GC,

    "all_contract_checks_passed":
        all_contract_checks_pass_628gc,
}


# ============================================================
# 22. HARD CLOSURE
# ============================================================

if not all_contract_checks_pass_628gc:

    failed_628gc = (
        FD002_CORRUPTION_SMOKE_AUDIT_628GC[
            ~FD002_CORRUPTION_SMOKE_AUDIT_628GC[
                "passed"
            ]
        ][
            "check"
        ]
        .tolist()
    )

    raise RuntimeError(
        "FD002 corruption smoke validation failed: "
        + str(
            failed_628gc
        )
    )


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-C FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Corruption operators defined : 5"
)

print(
    "Smoke-test sensor            :",
    SMOKE_SENSOR_628GC
)

print(
    "All operator checks passed   :",
    all_contract_checks_pass_628gc
)

print(
    "Clean DEV source unchanged   :",
    source_unchanged_628gc
)

print(
    "All smoke outputs finite     :",
    all_smoke_finite_628gc
)

print(
    "Magnitude scale source       : FD002 FIT only"
)

print(
    "Dropout imputation source    : FD002 FIT only"
)

print(
    "DEV labels used              : False"
)

print(
    "Official TEST labels read    : False"
)

print(
    "TEST scored                  : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Model retrained              : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28G-C ==="
)

In [ ]:
# ============================================================
# STEP 6.28G-D
# FREEZE COMPLETE FD002 CORRUPTION SCENARIO MANIFEST
#
# PURPOSE
# -------
# Construct the complete, deterministic FD002 scenario grid:
#
# bias    : 15 sensors × 3 severities × 2 directions = 90
# noise   : 15 sensors × 3 severities                = 45
# drift   : 15 sensors × 3 severities × 2 directions = 90
# dropout : 15 sensors × 3 severities                = 45
# stuck   : 15 sensors × 3 severities                = 45
#
# TOTAL = 315 scenarios
#
# IMPORTANT
# ---------
# - No corruption is applied.
# - No predictions are generated.
# - No official TEST labels are read.
# - No threshold is selected.
# - No model retraining.
# - No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-D — FREEZE FD002 CORRUPTION MANIFEST\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628gd = [

    "FD002_MODEL_SENSORS_628GB",

    "BIAS_SEVERITY_MAP_628GB",
    "NOISE_SEVERITY_MAP_628GB",
    "DRIFT_SEVERITY_MAP_628GB",
    "DROPOUT_SEVERITY_MAP_628GB",
    "STUCK_SEVERITY_MAP_628GB",

    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
    "FD002_CORRUPTION_SMOKE_AUDIT_628GC",
]


print(
    "\n--- REQUIRED OBJECTS ---"
)


missing_628gd = []


for name_628gd in required_628gd:

    exists_628gd = (
        name_628gd in globals()
    )

    print(
        f"{name_628gd:<48}: "
        f"{exists_628gd}"
    )

    if not exists_628gd:

        missing_628gd.append(
            name_628gd
        )


if missing_628gd:

    raise RuntimeError(
        "Missing required objects: "
        + str(
            missing_628gd
        )
    )


# ============================================================
# 2. PRECONDITION — SMOKE CONTRACT MUST HAVE PASSED
# ============================================================

all_smoke_passed_628gd = bool(

    FD002_CORRUPTION_SMOKE_AUDIT_628GC[
        "passed"
    ].all()
)


print(
    "\n--- PRECONDITION ---"
)

print(
    "All Step 6.28G-C smoke checks passed:",
    all_smoke_passed_628gd
)


if not all_smoke_passed_628gd:

    raise RuntimeError(
        "Cannot freeze scenario manifest because "
        "Step 6.28G-C did not pass completely."
    )


# ============================================================
# 3. FROZEN ORDERING CONTRACT
# ============================================================

SENSORS_628GD = list(
    FD002_MODEL_SENSORS_628GB
)


SEVERITY_ORDER_628GD = [
    "low",
    "medium",
    "high",
]


DIRECTION_ORDER_628GD = [
    -1,
    +1,
]


NOISE_SEED_628GD = 42

DROPOUT_SEED_628GD = 42


print(
    "\n--- FROZEN ORDERING CONTRACT ---"
)

print(
    "Sensors:",
    len(
        SENSORS_628GD
    )
)

print(
    "Sensor order:",
    SENSORS_628GD
)

print(
    "Severity order:",
    SEVERITY_ORDER_628GD
)

print(
    "Direction order:",
    DIRECTION_ORDER_628GD
)


if len(
    SENSORS_628GD
) != 15:

    raise RuntimeError(
        "Expected exactly 15 retained sensors."
    )


if len(
    set(
        SENSORS_628GD
    )
) != 15:

    raise RuntimeError(
        "Sensor list contains duplicates."
    )


# ============================================================
# 4. BUILD MANIFEST ROWS
# ============================================================

rows_628gd = []

scenario_counter_628gd = 1


# ------------------------------------------------------------
# BIAS
# ------------------------------------------------------------

for sensor_628gd in SENSORS_628GD:

    for severity_628gd in SEVERITY_ORDER_628GD:

        alpha_628gd = float(
            BIAS_SEVERITY_MAP_628GB[
                severity_628gd
            ]
        )

        for direction_628gd in DIRECTION_ORDER_628GD:

            rows_628gd.append(
                {
                    "scenario_number":
                        scenario_counter_628gd,

                    "family":
                        "bias",

                    "sensor":
                        sensor_628gd,

                    "severity":
                        severity_628gd,

                    "alpha":
                        alpha_628gd,

                    "direction":
                        int(
                            direction_628gd
                        ),

                    "requested_fraction":
                        np.nan,

                    "seed":
                        np.nan,
                }
            )

            scenario_counter_628gd += 1


# ------------------------------------------------------------
# NOISE
# ------------------------------------------------------------

for sensor_628gd in SENSORS_628GD:

    for severity_628gd in SEVERITY_ORDER_628GD:

        alpha_628gd = float(
            NOISE_SEVERITY_MAP_628GB[
                severity_628gd
            ]
        )

        rows_628gd.append(
            {
                "scenario_number":
                    scenario_counter_628gd,

                "family":
                    "noise",

                "sensor":
                    sensor_628gd,

                "severity":
                    severity_628gd,

                "alpha":
                    alpha_628gd,

                "direction":
                    np.nan,

                "requested_fraction":
                    np.nan,

                "seed":
                    int(
                        NOISE_SEED_628GD
                    ),
            }
        )

        scenario_counter_628gd += 1


# ------------------------------------------------------------
# DRIFT
# ------------------------------------------------------------

for sensor_628gd in SENSORS_628GD:

    for severity_628gd in SEVERITY_ORDER_628GD:

        alpha_628gd = float(
            DRIFT_SEVERITY_MAP_628GB[
                severity_628gd
            ]
        )

        for direction_628gd in DIRECTION_ORDER_628GD:

            rows_628gd.append(
                {
                    "scenario_number":
                        scenario_counter_628gd,

                    "family":
                        "drift",

                    "sensor":
                        sensor_628gd,

                    "severity":
                        severity_628gd,

                    "alpha":
                        alpha_628gd,

                    "direction":
                        int(
                            direction_628gd
                        ),

                    "requested_fraction":
                        np.nan,

                    "seed":
                        np.nan,
                }
            )

            scenario_counter_628gd += 1


# ------------------------------------------------------------
# DROPOUT
# ------------------------------------------------------------

for sensor_628gd in SENSORS_628GD:

    for severity_628gd in SEVERITY_ORDER_628GD:

        fraction_628gd = float(
            DROPOUT_SEVERITY_MAP_628GB[
                severity_628gd
            ]
        )

        rows_628gd.append(
            {
                "scenario_number":
                    scenario_counter_628gd,

                "family":
                    "dropout",

                "sensor":
                    sensor_628gd,

                "severity":
                    severity_628gd,

                "alpha":
                    np.nan,

                "direction":
                    np.nan,

                "requested_fraction":
                    fraction_628gd,

                "seed":
                    int(
                        DROPOUT_SEED_628GD
                    ),
            }
        )

        scenario_counter_628gd += 1


# ------------------------------------------------------------
# STUCK
# ------------------------------------------------------------

for sensor_628gd in SENSORS_628GD:

    for severity_628gd in SEVERITY_ORDER_628GD:

        fraction_628gd = float(
            STUCK_SEVERITY_MAP_628GB[
                severity_628gd
            ]
        )

        rows_628gd.append(
            {
                "scenario_number":
                    scenario_counter_628gd,

                "family":
                    "stuck",

                "sensor":
                    sensor_628gd,

                "severity":
                    severity_628gd,

                "alpha":
                    np.nan,

                "direction":
                    np.nan,

                "requested_fraction":
                    fraction_628gd,

                "seed":
                    np.nan,
            }
        )

        scenario_counter_628gd += 1


# ============================================================
# 5. CREATE MANIFEST
# ============================================================

FD002_CORRUPTION_MANIFEST_628GD = (
    pd.DataFrame(
        rows_628gd
    )
)


FD002_CORRUPTION_MANIFEST_628GD[
    "scenario_id"
] = (

    FD002_CORRUPTION_MANIFEST_628GD[
        "scenario_number"
    ]
    .map(
        lambda x_628gd:
            f"FD002_SCENARIO_{int(x_628gd):03d}"
    )
)


# Put scenario_id first.
column_order_628gd = [

    "scenario_id",
    "scenario_number",

    "family",
    "sensor",
    "severity",

    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]


FD002_CORRUPTION_MANIFEST_628GD = (

    FD002_CORRUPTION_MANIFEST_628GD[
        column_order_628gd
    ]
)


# ============================================================
# 6. BASIC STRUCTURAL CONTRACT
# ============================================================

total_scenarios_628gd = int(
    len(
        FD002_CORRUPTION_MANIFEST_628GD
    )
)


unique_ids_628gd = int(

    FD002_CORRUPTION_MANIFEST_628GD[
        "scenario_id"
    ].nunique()
)


duplicate_ids_628gd = int(

    FD002_CORRUPTION_MANIFEST_628GD[
        "scenario_id"
    ].duplicated().sum()
)


duplicate_semantic_rows_628gd = int(

    FD002_CORRUPTION_MANIFEST_628GD[
        [
            "family",
            "sensor",
            "severity",
            "alpha",
            "direction",
            "requested_fraction",
            "seed",
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\n--- STRUCTURAL CONTRACT ---"
)

print(
    "Total scenarios        :",
    total_scenarios_628gd
)

print(
    "Unique scenario IDs    :",
    unique_ids_628gd
)

print(
    "Duplicate IDs          :",
    duplicate_ids_628gd
)

print(
    "Duplicate semantic rows:",
    duplicate_semantic_rows_628gd
)


if total_scenarios_628gd != 315:

    raise RuntimeError(
        "Expected exactly 315 scenarios."
    )


if unique_ids_628gd != 315:

    raise RuntimeError(
        "Scenario IDs are not unique."
    )


if duplicate_ids_628gd != 0:

    raise RuntimeError(
        "Duplicate scenario IDs detected."
    )


if duplicate_semantic_rows_628gd != 0:

    raise RuntimeError(
        "Duplicate semantic scenarios detected."
    )


# ============================================================
# 7. FAMILY COUNT CONTRACT
# ============================================================

family_counts_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD[
        "family"
    ]
    .value_counts()
    .sort_index()
)


expected_family_counts_628gd = {

    "bias":
        90,

    "noise":
        45,

    "drift":
        90,

    "dropout":
        45,

    "stuck":
        45,
}


print(
    "\n--- FAMILY COUNTS ---"
)

print(
    family_counts_628gd
)


for family_628gd, expected_628gd in (
    expected_family_counts_628gd.items()
):

    actual_628gd = int(
        family_counts_628gd.get(
            family_628gd,
            0
        )
    )

    if actual_628gd != expected_628gd:

        raise RuntimeError(
            f"{family_628gd}: expected "
            f"{expected_628gd}, got "
            f"{actual_628gd}."
        )


# ============================================================
# 8. FAMILY × SEVERITY CONTRACT
# ============================================================

family_severity_counts_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD

    .groupby(
        [
            "family",
            "severity",
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )

    .reindex(
        columns=
            SEVERITY_ORDER_628GD
    )
)


print(
    "\n--- FAMILY × SEVERITY COUNTS ---"
)

display(
    family_severity_counts_628gd
)


expected_family_severity_628gd = {

    "bias":
        30,

    "noise":
        15,

    "drift":
        30,

    "dropout":
        15,

    "stuck":
        15,
}


for family_628gd, expected_each_628gd in (
    expected_family_severity_628gd.items()
):

    for severity_628gd in (
        SEVERITY_ORDER_628GD
    ):

        actual_628gd = int(
            family_severity_counts_628gd.loc[
                family_628gd,
                severity_628gd
            ]
        )

        if actual_628gd != expected_each_628gd:

            raise RuntimeError(
                f"{family_628gd}/{severity_628gd}: "
                f"expected {expected_each_628gd}, "
                f"got {actual_628gd}."
            )


# ============================================================
# 9. SENSOR-COVERAGE CONTRACT
# ============================================================

sensor_family_counts_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD

    .groupby(
        [
            "family",
            "sensor",
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )
)


expected_per_sensor_628gd = {

    "bias":
        6,

    "noise":
        3,

    "drift":
        6,

    "dropout":
        3,

    "stuck":
        3,
}


sensor_contract_ok_628gd = True


for family_628gd, expected_628gd in (
    expected_per_sensor_628gd.items()
):

    family_rows_628gd = (

        FD002_CORRUPTION_MANIFEST_628GD[
            FD002_CORRUPTION_MANIFEST_628GD[
                "family"
            ]
            ==
            family_628gd
        ]
    )


    observed_sensors_628gd = set(
        family_rows_628gd[
            "sensor"
        ].unique()
    )


    if observed_sensors_628gd != set(
        SENSORS_628GD
    ):

        sensor_contract_ok_628gd = False


    counts_628gd = (
        family_rows_628gd
        .groupby(
            "sensor"
        )
        .size()
    )


    if not (
        counts_628gd
        ==
        expected_628gd
    ).all():

        sensor_contract_ok_628gd = False


print(
    "\n--- SENSOR COVERAGE CONTRACT ---"
)

print(
    "All 15 sensors present in every family:",
    sensor_contract_ok_628gd
)


if not sensor_contract_ok_628gd:

    raise RuntimeError(
        "Sensor coverage contract failed."
    )


# ============================================================
# 10. PARAMETER-MAPPING VALIDATION
# ============================================================

parameter_mapping_ok_628gd = True


mapping_contract_628gd = {

    "bias":
        (
            "alpha",
            BIAS_SEVERITY_MAP_628GB
        ),

    "noise":
        (
            "alpha",
            NOISE_SEVERITY_MAP_628GB
        ),

    "drift":
        (
            "alpha",
            DRIFT_SEVERITY_MAP_628GB
        ),

    "dropout":
        (
            "requested_fraction",
            DROPOUT_SEVERITY_MAP_628GB
        ),

    "stuck":
        (
            "requested_fraction",
            STUCK_SEVERITY_MAP_628GB
        ),
}


for family_628gd, (
    parameter_column_628gd,
    severity_map_628gd
) in mapping_contract_628gd.items():

    family_df_628gd = (

        FD002_CORRUPTION_MANIFEST_628GD[
            FD002_CORRUPTION_MANIFEST_628GD[
                "family"
            ]
            ==
            family_628gd
        ]
    )


    for severity_628gd in (
        SEVERITY_ORDER_628GD
    ):

        expected_value_628gd = float(
            severity_map_628gd[
                severity_628gd
            ]
        )


        observed_values_628gd = (

            family_df_628gd[
                family_df_628gd[
                    "severity"
                ]
                ==
                severity_628gd
            ][
                parameter_column_628gd
            ]
            .dropna()
            .unique()
        )


        exact_628gd = (

            len(
                observed_values_628gd
            )
            ==
            1

            and

            np.isclose(
                float(
                    observed_values_628gd[
                        0
                    ]
                ),
                expected_value_628gd,
                atol=0.0,
                rtol=0.0
            )
        )


        if not exact_628gd:

            parameter_mapping_ok_628gd = False


print(
    "\n--- PARAMETER MAPPING ---"
)

print(
    "All severity mappings exact:",
    parameter_mapping_ok_628gd
)


if not parameter_mapping_ok_628gd:

    raise RuntimeError(
        "Severity mapping validation failed."
    )


# ============================================================
# 11. DIRECTION CONTRACT
# ============================================================

bias_directions_628gd = sorted(

    FD002_CORRUPTION_MANIFEST_628GD.loc[
        FD002_CORRUPTION_MANIFEST_628GD[
            "family"
        ]
        ==
        "bias",
        "direction"
    ]
    .dropna()
    .unique()
    .tolist()
)


drift_directions_628gd = sorted(

    FD002_CORRUPTION_MANIFEST_628GD.loc[
        FD002_CORRUPTION_MANIFEST_628GD[
            "family"
        ]
        ==
        "drift",
        "direction"
    ]
    .dropna()
    .unique()
    .tolist()
)


direction_contract_ok_628gd = (

    bias_directions_628gd
    ==
    [
        -1.0,
        1.0,
    ]

    and

    drift_directions_628gd
    ==
    [
        -1.0,
        1.0,
    ]
)


print(
    "\n--- DIRECTION CONTRACT ---"
)

print(
    "Bias directions :",
    bias_directions_628gd
)

print(
    "Drift directions:",
    drift_directions_628gd
)

print(
    "Direction contract exact:",
    direction_contract_ok_628gd
)


if not direction_contract_ok_628gd:

    raise RuntimeError(
        "Direction contract failed."
    )


# ============================================================
# 12. SEED CONTRACT
# ============================================================

noise_seeds_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD.loc[
        FD002_CORRUPTION_MANIFEST_628GD[
            "family"
        ]
        ==
        "noise",
        "seed"
    ]
    .dropna()
    .unique()
)


dropout_seeds_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD.loc[
        FD002_CORRUPTION_MANIFEST_628GD[
            "family"
        ]
        ==
        "dropout",
        "seed"
    ]
    .dropna()
    .unique()
)


seed_contract_ok_628gd = bool(

    len(
        noise_seeds_628gd
    )
    ==
    1

    and

    len(
        dropout_seeds_628gd
    )
    ==
    1

    and

    int(
        noise_seeds_628gd[
            0
        ]
    )
    ==
    42

    and

    int(
        dropout_seeds_628gd[
            0
        ]
    )
    ==
    42
)


print(
    "\n--- SEED CONTRACT ---"
)

print(
    "Noise seeds:",
    noise_seeds_628gd.tolist()
)

print(
    "Dropout seeds:",
    dropout_seeds_628gd.tolist()
)

print(
    "Seed contract exact:",
    seed_contract_ok_628gd
)


if not seed_contract_ok_628gd:

    raise RuntimeError(
        "Seed contract failed."
    )


# ============================================================
# 13. SCENARIO NUMBER CONTINUITY
# ============================================================

expected_numbers_628gd = np.arange(
    1,
    316,
    dtype=np.int64
)


actual_numbers_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD[
        "scenario_number"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


scenario_numbers_exact_628gd = bool(
    np.array_equal(
        expected_numbers_628gd,
        actual_numbers_628gd
    )
)


print(
    "\n--- SCENARIO NUMBER CONTRACT ---"
)

print(
    "Continuous 001–315:",
    scenario_numbers_exact_628gd
)


if not scenario_numbers_exact_628gd:

    raise RuntimeError(
        "Scenario numbering is not exact."
    )


# ============================================================
# 14. MANIFEST PREVIEW
# ============================================================

print(
    "\n--- FIRST 12 SCENARIOS ---"
)

display(
    FD002_CORRUPTION_MANIFEST_628GD
    .head(
        12
    )
)


print(
    "\n--- LAST 12 SCENARIOS ---"
)

display(
    FD002_CORRUPTION_MANIFEST_628GD
    .tail(
        12
    )
)


# ============================================================
# 15. FAMILY BOUNDARY AUDIT
# ============================================================

family_boundaries_628gd = (

    FD002_CORRUPTION_MANIFEST_628GD

    .groupby(
        "family",
        sort=False
    )

    .agg(

        first_scenario=(
            "scenario_number",
            "min"
        ),

        last_scenario=(
            "scenario_number",
            "max"
        ),

        scenarios=(
            "scenario_number",
            "size"
        ),

        sensors=(
            "sensor",
            "nunique"
        ),
    )

    .reset_index()
)


print(
    "\n--- FAMILY BOUNDARIES ---"
)

display(
    family_boundaries_628gd
)


# ============================================================
# 16. FREEZE MANIFEST CONTRACT
# ============================================================

FD002_CORRUPTION_MANIFEST_CONTRACT_628GD = {

    "dataset":
        "FD002",

    "total_scenarios":
        315,

    "families":
        {
            "bias":
                90,

            "noise":
                45,

            "drift":
                90,

            "dropout":
                45,

            "stuck":
                45,
        },

    "sensors":
        SENSORS_628GD.copy(),

    "sensor_count":
        15,

    "severity_order":
        SEVERITY_ORDER_628GD.copy(),

    "direction_order":
        DIRECTION_ORDER_628GD.copy(),

    "noise_seed":
        42,

    "dropout_seed":
        42,

    "magnitude_reference":
        (
            "FD002 FIT-only pooled "
            "within-operating-regime STD"
        ),

    "dropout_imputation":
        (
            "FD002 FIT-only "
            "regime-specific sensor mean"
        ),

    "corruption_space":
        "raw sensor measurement space",

    "official_test_labels_used":
        False,

    "all_contract_checks_passed":
        True,
}


# ============================================================
# 17. FINAL VALIDATION
# ============================================================

final_checks_628gd = {

    "scenario_count_315":
        (
            total_scenarios_628gd
            ==
            315
        ),

    "scenario_ids_unique":
        (
            unique_ids_628gd
            ==
            315
        ),

    "no_duplicate_semantics":
        (
            duplicate_semantic_rows_628gd
            ==
            0
        ),

    "family_counts_exact":
        all(
            int(
                family_counts_628gd.get(
                    family_628gd,
                    0
                )
            )
            ==
            expected_628gd

            for (
                family_628gd,
                expected_628gd
            )

            in expected_family_counts_628gd.items()
        ),

    "sensor_coverage_exact":
        sensor_contract_ok_628gd,

    "parameter_mapping_exact":
        parameter_mapping_ok_628gd,

    "direction_contract_exact":
        direction_contract_ok_628gd,

    "seed_contract_exact":
        seed_contract_ok_628gd,

    "scenario_numbers_exact":
        scenario_numbers_exact_628gd,
}


FD002_CORRUPTION_MANIFEST_AUDIT_628GD = (

    pd.DataFrame(
        [
            {
                "check":
                    key_628gd,

                "passed":
                    bool(
                        value_628gd
                    ),
            }

            for (
                key_628gd,
                value_628gd
            )

            in final_checks_628gd.items()
        ]
    )
)


print(
    "\n--- FINAL MANIFEST AUDIT ---"
)

display(
    FD002_CORRUPTION_MANIFEST_AUDIT_628GD
)


all_manifest_checks_pass_628gd = bool(

    FD002_CORRUPTION_MANIFEST_AUDIT_628GD[
        "passed"
    ].all()
)


if not all_manifest_checks_pass_628gd:

    failed_checks_628gd = (

        FD002_CORRUPTION_MANIFEST_AUDIT_628GD[
            ~FD002_CORRUPTION_MANIFEST_AUDIT_628GD[
                "passed"
            ]
        ][
            "check"
        ]
        .tolist()
    )

    raise RuntimeError(
        "Manifest validation failed: "
        + str(
            failed_checks_628gd
        )
    )


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28G-D FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Total frozen scenarios       :",
    total_scenarios_628gd
)

print(
    "Bias scenarios               :",
    int(
        family_counts_628gd[
            "bias"
        ]
    )
)

print(
    "Noise scenarios              :",
    int(
        family_counts_628gd[
            "noise"
        ]
    )
)

print(
    "Drift scenarios              :",
    int(
        family_counts_628gd[
            "drift"
        ]
    )
)

print(
    "Dropout scenarios            :",
    int(
        family_counts_628gd[
            "dropout"
        ]
    )
)

print(
    "Stuck scenarios              :",
    int(
        family_counts_628gd[
            "stuck"
        ]
    )
)

print(
    "Retained sensors             :",
    len(
        SENSORS_628GD
    )
)

print(
    "All manifest checks passed   :",
    all_manifest_checks_pass_628gd
)

print(
    "Corruption applied           : False"
)

print(
    "Predictions generated        : False"
)

print(
    "DEV labels used              : False"
)

print(
    "Official TEST labels read    : False"
)

print(
    "TEST scored                  : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Model retrained              : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28G-D ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-A
# FD002 CORRUPTION -> PREPROCESSING -> GRU BRIDGE AUDIT
#
# PURPOSE
# -------
# Before running 315 corruption scenarios, verify that we can:
#
#   raw DEV
#       -> frozen FIT-only regime normalization
#       -> 30-cycle window reconstruction
#       -> trained GRU
#
# and reproduce the already-frozen clean DEV result.
#
# Then run ONE label-free corruption scenario through the same
# bridge as a smoke test.
#
# NO official TEST labels.
# NO TEST scoring.
# NO threshold selection.
# NO model retraining.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd
import torch


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-A — CORRUPTION / MODEL-BRIDGE AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628ha = [

    "FD002_FIT_TARGETED_628B",
    "FD002_FIT_NORMALIZED_628C",

    "FD002_DEV_TARGETED_628B",
    "FD002_DEV_NORMALIZED_628C",

    "FD002_MODEL_FEATURES_628C",
    "FD002_MODEL_SENSORS_628GB",

    "FD002_WINDOW_LENGTH_628B",

    "DEV_WINDOW_KEYS_FD002_628D",
    "X_DEV_FD002_628D",

    "MODEL_FD002_628EB",
    "DEV_PREDICTIONS_FD002_628EB",

    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",

    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_MANIFEST_CONTRACT_628GD",
]


print("\n--- REQUIRED OBJECTS ---")

missing_628ha = []

for name_628ha in required_628ha:

    exists_628ha = name_628ha in globals()

    print(
        f"{name_628ha:<48}: "
        f"{exists_628ha}"
    )

    if not exists_628ha:
        missing_628ha.append(name_628ha)


if missing_628ha:

    raise RuntimeError(
        "Missing required objects: "
        + str(missing_628ha)
    )


# ============================================================
# 2. FREEZE LOCAL REFERENCES
# ============================================================

fit_raw_628ha = (
    FD002_FIT_TARGETED_628B
    .copy()
    .reset_index(drop=True)
)

fit_norm_628ha = (
    FD002_FIT_NORMALIZED_628C
    .copy()
    .reset_index(drop=True)
)

dev_raw_628ha = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .copy()
    .reset_index(drop=True)
)

dev_norm_628ha = (
    FD002_DEV_NORMALIZED_628C
    .copy()
    .reset_index(drop=True)
)

dev_window_keys_628ha = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)

model_features_628ha = list(
    FD002_MODEL_FEATURES_628C
)

model_sensors_628ha = list(
    FD002_MODEL_SENSORS_628GB
)

window_length_628ha = int(
    FD002_WINDOW_LENGTH_628B
)


print("\n--- FROZEN BRIDGE CONTRACT ---")

print(
    "FIT raw rows       :",
    len(fit_raw_628ha)
)

print(
    "DEV raw rows       :",
    len(dev_raw_628ha)
)

print(
    "DEV window rows    :",
    len(dev_window_keys_628ha)
)

print(
    "Window length      :",
    window_length_628ha
)

print(
    "Model feature count:",
    len(model_features_628ha)
)

print(
    "Model sensor count :",
    len(model_sensors_628ha)
)


if window_length_628ha != 30:
    raise RuntimeError(
        "Frozen window length is not 30."
    )


if len(model_features_628ha) != 17:
    raise RuntimeError(
        "Frozen model feature count is not 17."
    )


if len(model_sensors_628ha) != 15:
    raise RuntimeError(
        "Frozen sensor count is not 15."
    )


# ============================================================
# 3. RAW / NORMALIZED ROW-ALIGNMENT AUDIT
# ============================================================

key_cols_628ha = [
    "unit_id",
    "cycle",
]


for col_628ha in key_cols_628ha:

    if col_628ha not in fit_raw_628ha.columns:
        raise RuntimeError(
            f"{col_628ha} missing from FIT raw."
        )

    if col_628ha not in fit_norm_628ha.columns:
        raise RuntimeError(
            f"{col_628ha} missing from FIT normalized."
        )

    if col_628ha not in dev_raw_628ha.columns:
        raise RuntimeError(
            f"{col_628ha} missing from DEV raw."
        )

    if col_628ha not in dev_norm_628ha.columns:
        raise RuntimeError(
            f"{col_628ha} missing from DEV normalized."
        )


fit_alignment_628ha = bool(

    fit_raw_628ha[
        key_cols_628ha
    ].equals(

        fit_norm_628ha[
            key_cols_628ha
        ]
    )
)


dev_alignment_628ha = bool(

    dev_raw_628ha[
        key_cols_628ha
    ].equals(

        dev_norm_628ha[
            key_cols_628ha
        ]
    )
)


print("\n--- ROW ALIGNMENT ---")

print(
    "FIT exact alignment:",
    fit_alignment_628ha
)

print(
    "DEV exact alignment:",
    dev_alignment_628ha
)


if not fit_alignment_628ha:
    raise RuntimeError(
        "FIT raw/normalized alignment failed."
    )

if not dev_alignment_628ha:
    raise RuntimeError(
        "DEV raw/normalized alignment failed."
    )


if "regime_id" not in fit_norm_628ha.columns:
    raise RuntimeError(
        "FIT normalized table has no regime_id."
    )

if "regime_id" not in dev_norm_628ha.columns:
    raise RuntimeError(
        "DEV normalized table has no regime_id."
    )


# ============================================================
# 4. IDENTIFY EXACT SENSOR-STD CONVENTION
#
# We independently test ddof=0 and ddof=1 against the already
# frozen normalized FIT data.
#
# We do NOT assume the answer.
# ============================================================

def normalization_error_for_ddof_628ha(
    ddof
):

    max_error_628ha = 0.0

    fit_regimes_628ha = (
        fit_norm_628ha[
            "regime_id"
        ]
        .to_numpy()
    )

    for sensor_628ha in model_sensors_628ha:

        normalized_col_628ha = (
            f"{sensor_628ha}_z"
        )

        if normalized_col_628ha not in (
            fit_norm_628ha.columns
        ):

            raise RuntimeError(
                f"{normalized_col_628ha} "
                "missing from normalized FIT."
            )


        reconstructed_628ha = np.empty(
            len(fit_raw_628ha),
            dtype=np.float64
        )


        for regime_628ha in np.unique(
            fit_regimes_628ha
        ):

            mask_628ha = (
                fit_regimes_628ha
                ==
                regime_628ha
            )


            raw_values_628ha = (
                fit_raw_628ha.loc[
                    mask_628ha,
                    sensor_628ha
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            mean_628ha = float(
                np.mean(
                    raw_values_628ha
                )
            )


            std_628ha = float(
                np.std(
                    raw_values_628ha,
                    ddof=ddof
                )
            )


            if (
                not np.isfinite(std_628ha)
                or
                std_628ha <= 0.0
            ):

                raise RuntimeError(
                    "Invalid regime sensor STD: "
                    f"sensor={sensor_628ha}, "
                    f"regime={regime_628ha}, "
                    f"ddof={ddof}"
                )


            reconstructed_628ha[
                mask_628ha
            ] = (
                raw_values_628ha
                -
                mean_628ha
            ) / std_628ha


        frozen_628ha = (
            fit_norm_628ha[
                normalized_col_628ha
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        sensor_error_628ha = float(
            np.max(
                np.abs(
                    reconstructed_628ha
                    -
                    frozen_628ha
                )
            )
        )


        max_error_628ha = max(
            max_error_628ha,
            sensor_error_628ha
        )


    return max_error_628ha


ddof_errors_628ha = {

    0:
        normalization_error_for_ddof_628ha(
            0
        ),

    1:
        normalization_error_for_ddof_628ha(
            1
        ),
}


NORMALIZATION_DDOF_628HA = min(
    ddof_errors_628ha,
    key=ddof_errors_628ha.get
)


best_fit_normalization_error_628ha = float(
    ddof_errors_628ha[
        NORMALIZATION_DDOF_628HA
    ]
)


print(
    "\n--- NORMALIZATION-CONVENTION AUDIT ---"
)

print(
    "ddof=0 max FIT error:",
    ddof_errors_628ha[0]
)

print(
    "ddof=1 max FIT error:",
    ddof_errors_628ha[1]
)

print(
    "Resolved ddof       :",
    NORMALIZATION_DDOF_628HA
)

print(
    "Best max FIT error  :",
    best_fit_normalization_error_628ha
)


if best_fit_normalization_error_628ha > 1e-8:

    raise RuntimeError(
        "Could not reproduce frozen FIT "
        "sensor normalization accurately."
    )


# ============================================================
# 5. RECONSTRUCT FIT-ONLY REGIME × SENSOR STATISTICS
# ============================================================

fit_regimes_628ha = (
    fit_norm_628ha[
        "regime_id"
    ]
    .to_numpy()
)


stats_rows_628ha = []


for regime_628ha in sorted(
    np.unique(
        fit_regimes_628ha
    )
):

    regime_mask_628ha = (
        fit_regimes_628ha
        ==
        regime_628ha
    )


    for sensor_628ha in model_sensors_628ha:

        values_628ha = (
            fit_raw_628ha.loc[
                regime_mask_628ha,
                sensor_628ha
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        stats_rows_628ha.append(
            {
                "regime_id":
                    int(regime_628ha),

                "sensor":
                    sensor_628ha,

                "mean":
                    float(
                        np.mean(
                            values_628ha
                        )
                    ),

                "std":
                    float(
                        np.std(
                            values_628ha,
                            ddof=
                                NORMALIZATION_DDOF_628HA
                        )
                    ),

                "fit_rows":
                    int(
                        len(
                            values_628ha
                        )
                    ),
            }
        )


FD002_RECONSTRUCTED_REGIME_SENSOR_STATS_628HA = (
    pd.DataFrame(
        stats_rows_628ha
    )
)


print(
    "\n--- RECONSTRUCTED FIT-ONLY SENSOR STATISTICS ---"
)

print(
    "Rows:",
    len(
        FD002_RECONSTRUCTED_REGIME_SENSOR_STATS_628HA
    )
)

print(
    "Expected:",
    6 * 15
)


if len(
    FD002_RECONSTRUCTED_REGIME_SENSOR_STATS_628HA
) != 90:

    raise RuntimeError(
        "Expected 90 regime × sensor "
        "normalization rows."
    )


# Fast lookup.
REGIME_SENSOR_STATS_628HA = {

    (
        int(row_628ha["regime_id"]),
        str(row_628ha["sensor"])
    ):
        (
            float(row_628ha["mean"]),
            float(row_628ha["std"])
        )

    for _, row_628ha in
    FD002_RECONSTRUCTED_REGIME_SENSOR_STATS_628HA.iterrows()
}


# ============================================================
# 6. VERIFY RECONSTRUCTED NORMALIZATION ON CLEAN DEV
# ============================================================

dev_regimes_628ha = (
    dev_norm_628ha[
        "regime_id"
    ]
    .to_numpy()
)


dev_normalization_errors_628ha = []


for sensor_628ha in model_sensors_628ha:

    reconstructed_628ha = np.empty(
        len(dev_raw_628ha),
        dtype=np.float64
    )


    raw_values_all_628ha = (
        dev_raw_628ha[
            sensor_628ha
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    for regime_628ha in np.unique(
        dev_regimes_628ha
    ):

        mask_628ha = (
            dev_regimes_628ha
            ==
            regime_628ha
        )


        mean_628ha, std_628ha = (
            REGIME_SENSOR_STATS_628HA[
                (
                    int(regime_628ha),
                    sensor_628ha
                )
            ]
        )


        reconstructed_628ha[
            mask_628ha
        ] = (

            raw_values_all_628ha[
                mask_628ha
            ]
            -
            mean_628ha

        ) / std_628ha


    frozen_628ha = (
        dev_norm_628ha[
            f"{sensor_628ha}_z"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    max_error_628ha = float(
        np.max(
            np.abs(
                reconstructed_628ha
                -
                frozen_628ha
            )
        )
    )


    dev_normalization_errors_628ha.append(
        {
            "sensor":
                sensor_628ha,

            "max_abs_error":
                max_error_628ha,
        }
    )


FD002_DEV_NORMALIZATION_REPRODUCTION_628HA = (
    pd.DataFrame(
        dev_normalization_errors_628ha
    )
)


max_dev_normalization_error_628ha = float(

    FD002_DEV_NORMALIZATION_REPRODUCTION_628HA[
        "max_abs_error"
    ].max()
)


print(
    "\n--- CLEAN DEV NORMALIZATION REPRODUCTION ---"
)

print(
    "Maximum absolute error:",
    max_dev_normalization_error_628ha
)


if max_dev_normalization_error_628ha > 1e-8:

    display(
        FD002_DEV_NORMALIZATION_REPRODUCTION_628HA
        .sort_values(
            "max_abs_error",
            ascending=False
        )
    )

    raise RuntimeError(
        "Clean DEV normalization "
        "was not reproduced."
    )


# ============================================================
# 7. CORRUPTED-SENSOR NORMALIZATION HELPER
#
# All other normalized features remain exactly as frozen.
#
# Operating settings are not corrupted, therefore regime_id
# remains unchanged.
# ============================================================

def normalize_corrupted_sensor_628ha(
    corrupted_raw,
    sensor
):

    if sensor not in model_sensors_628ha:

        raise ValueError(
            f"Invalid model sensor: {sensor}"
        )


    corrupted_raw = (
        corrupted_raw
        .copy()
        .reset_index(drop=True)
    )


    if not corrupted_raw[
        key_cols_628ha
    ].equals(
        dev_norm_628ha[
            key_cols_628ha
        ]
    ):

        raise RuntimeError(
            "Corrupted raw DEV keys changed."
        )


    normalized_628ha = (
        dev_norm_628ha.copy()
    )


    raw_values_628ha = (
        corrupted_raw[
            sensor
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    z_values_628ha = np.empty(
        len(
            corrupted_raw
        ),
        dtype=np.float64
    )


    regime_ids_628ha = (
        dev_norm_628ha[
            "regime_id"
        ]
        .to_numpy()
    )


    for regime_628ha in np.unique(
        regime_ids_628ha
    ):

        mask_628ha = (
            regime_ids_628ha
            ==
            regime_628ha
        )


        mean_628ha, std_628ha = (
            REGIME_SENSOR_STATS_628HA[
                (
                    int(regime_628ha),
                    sensor
                )
            ]
        )


        z_values_628ha[
            mask_628ha
        ] = (

            raw_values_628ha[
                mask_628ha
            ]
            -
            mean_628ha

        ) / std_628ha


    normalized_628ha[
        f"{sensor}_z"
    ] = z_values_628ha


    return normalized_628ha


# ============================================================
# 8. WINDOW RECONSTRUCTION HELPER
# ============================================================

def build_dev_windows_628ha(
    normalized_df
):

    required_cols_628ha = (
        [
            "unit_id",
            "cycle",
        ]
        +
        model_features_628ha
    )


    missing_cols_628ha = [

        c_628ha

        for c_628ha
        in required_cols_628ha

        if c_628ha not in normalized_df.columns
    ]


    if missing_cols_628ha:

        raise RuntimeError(
            "Missing normalized columns: "
            + str(missing_cols_628ha)
        )


    engine_cache_628ha = {}


    for unit_id_628ha, group_628ha in (

        normalized_df
        .groupby(
            "unit_id",
            sort=False
        )
    ):

        group_628ha = (
            group_628ha
            .sort_values(
                "cycle"
            )
        )


        cycles_628ha = (
            group_628ha[
                "cycle"
            ]
            .to_numpy()
        )


        features_628ha = (
            group_628ha[
                model_features_628ha
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        engine_cache_628ha[
            unit_id_628ha
        ] = (
            cycles_628ha,
            features_628ha
        )


    X_628ha = np.empty(
        (
            len(
                dev_window_keys_628ha
            ),
            window_length_628ha,
            len(
                model_features_628ha
            ),
        ),
        dtype=np.float32
    )


    for i_628ha, row_628ha in (
        dev_window_keys_628ha.iterrows()
    ):

        unit_id_628ha = row_628ha[
            "unit_id"
        ]

        endpoint_cycle_628ha = row_628ha[
            "cycle"
        ]


        if unit_id_628ha not in (
            engine_cache_628ha
        ):

            raise RuntimeError(
                f"Unknown DEV engine "
                f"{unit_id_628ha}."
            )


        cycles_628ha, features_628ha = (
            engine_cache_628ha[
                unit_id_628ha
            ]
        )


        endpoint_pos_628ha = int(
            np.searchsorted(
                cycles_628ha,
                endpoint_cycle_628ha
            )
        )


        if (
            endpoint_pos_628ha
            >=
            len(cycles_628ha)
            or
            cycles_628ha[
                endpoint_pos_628ha
            ]
            !=
            endpoint_cycle_628ha
        ):

            raise RuntimeError(
                "DEV endpoint cycle "
                "could not be resolved."
            )


        start_pos_628ha = (
            endpoint_pos_628ha
            -
            window_length_628ha
            +
            1
        )


        if start_pos_628ha < 0:

            raise RuntimeError(
                "DEV window does not contain "
                "30 natural cycles."
            )


        window_628ha = features_628ha[
            start_pos_628ha:
            endpoint_pos_628ha + 1
        ]


        if window_628ha.shape != (
            window_length_628ha,
            len(model_features_628ha)
        ):

            raise RuntimeError(
                "Unexpected DEV window shape."
            )


        X_628ha[
            i_628ha
        ] = window_628ha


    return X_628ha


# ============================================================
# 9. CLEAN WINDOW REPRODUCTION
# ============================================================

X_DEV_REBUILT_628HA = (
    build_dev_windows_628ha(
        dev_norm_628ha
    )
)


frozen_X_dev_628ha = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32
)


if (
    X_DEV_REBUILT_628HA.shape
    !=
    frozen_X_dev_628ha.shape
):

    raise RuntimeError(
        "Rebuilt DEV window tensor "
        "has wrong shape."
    )


clean_window_max_error_628ha = float(
    np.max(
        np.abs(
            X_DEV_REBUILT_628HA
            -
            frozen_X_dev_628ha
        )
    )
)


print(
    "\n--- CLEAN WINDOW REPRODUCTION ---"
)

print(
    "Frozen shape :",
    frozen_X_dev_628ha.shape
)

print(
    "Rebuilt shape:",
    X_DEV_REBUILT_628HA.shape
)

print(
    "Maximum absolute difference:",
    clean_window_max_error_628ha
)


if clean_window_max_error_628ha > 1e-7:

    raise RuntimeError(
        "Frozen clean DEV windows "
        "were not reproduced."
    )


# ============================================================
# 10. DETERMINISTIC GRU INFERENCE HELPER
#
# Keep the original DEV inference batch size = 256.
# ============================================================

def predict_gru_628ha(
    X,
    batch_size=256
):

    model_628ha = MODEL_FD002_628EB

    device_628ha = next(
        model_628ha.parameters()
    ).device


    model_628ha.eval()


    predictions_628ha = []


    with torch.no_grad():

        for start_628ha in range(
            0,
            len(X),
            int(batch_size)
        ):

            end_628ha = min(
                start_628ha
                +
                int(batch_size),

                len(X)
            )


            batch_628ha = torch.as_tensor(
                X[
                    start_628ha:
                    end_628ha
                ],
                dtype=torch.float32,
                device=device_628ha
            )


            output_628ha = (
                model_628ha(
                    batch_628ha
                )
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )


            predictions_628ha.append(
                output_628ha
            )


    return np.concatenate(
        predictions_628ha
    )


def prediction_vector_628ha(
    obj
):

    if isinstance(
        obj,
        pd.DataFrame
    ):

        if "prediction" in obj.columns:

            return (
                obj["prediction"]
                .to_numpy(
                    dtype=np.float64
                )
            )

        arr_628ha = obj.to_numpy()

    elif isinstance(
        obj,
        pd.Series
    ):

        return obj.to_numpy(
            dtype=np.float64
        )

    else:

        arr_628ha = np.asarray(
            obj
        )


    arr_628ha = np.asarray(
        arr_628ha,
        dtype=np.float64
    ).reshape(-1)


    return arr_628ha


# ============================================================
# 11. CLEAN PREDICTION REPRODUCTION
# ============================================================

CLEAN_DEV_PREDICTIONS_REBUILT_628HA = (
    predict_gru_628ha(
        X_DEV_REBUILT_628HA,
        batch_size=256
    )
)


frozen_dev_predictions_628ha = (
    prediction_vector_628ha(
        DEV_PREDICTIONS_FD002_628EB
    )
)


if (
    len(
        CLEAN_DEV_PREDICTIONS_REBUILT_628HA
    )
    !=
    len(
        frozen_dev_predictions_628ha
    )
):

    raise RuntimeError(
        "Frozen/rebuilt prediction lengths differ."
    )


clean_prediction_max_error_628ha = float(
    np.max(
        np.abs(
            CLEAN_DEV_PREDICTIONS_REBUILT_628HA
            -
            frozen_dev_predictions_628ha
        )
    )
)


clean_prediction_mean_error_628ha = float(
    np.mean(
        np.abs(
            CLEAN_DEV_PREDICTIONS_REBUILT_628HA
            -
            frozen_dev_predictions_628ha
        )
    )
)


print(
    "\n--- CLEAN GRU PREDICTION REPRODUCTION ---"
)

print(
    "Prediction count :",
    len(
        CLEAN_DEV_PREDICTIONS_REBUILT_628HA
    )
)

print(
    "Maximum abs error:",
    clean_prediction_max_error_628ha
)

print(
    "Mean abs error   :",
    clean_prediction_mean_error_628ha
)


if clean_prediction_max_error_628ha > 2e-5:

    raise RuntimeError(
        "Frozen DEV predictions were "
        "not numerically reproduced."
    )


# ============================================================
# 12. GENERIC MANIFEST-SCENARIO DISPATCH
# ============================================================

def apply_manifest_scenario_628ha(
    base_raw,
    scenario_row
):

    family_628ha = str(
        scenario_row[
            "family"
        ]
    )

    sensor_628ha = str(
        scenario_row[
            "sensor"
        ]
    )


    contract_628ha = (
        FD002_CORRUPTION_OPERATOR_CONTRACT_628GC
    )


    if family_628ha == "bias":

        corrupted_628ha, _ = (
            contract_628ha[
                "bias"
            ](
                base_raw,
                sensor_628ha,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                )
            )
        )


    elif family_628ha == "noise":

        corrupted_628ha, _, _ = (
            contract_628ha[
                "noise"
            ](
                base_raw,
                sensor_628ha,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                )
            )
        )


    elif family_628ha == "drift":

        corrupted_628ha, _, _ = (
            contract_628ha[
                "drift"
            ](
                base_raw,
                sensor_628ha,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                )
            )
        )


    elif family_628ha == "dropout":

        corrupted_628ha, _, _ = (
            contract_628ha[
                "dropout"
            ](
                base_raw,
                sensor_628ha,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                )
            )
        )


    elif family_628ha == "stuck":

        corrupted_628ha, _, _ = (
            contract_628ha[
                "stuck"
            ](
                base_raw,
                sensor_628ha,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                )
            )
        )


    else:

        raise ValueError(
            "Unknown corruption family: "
            + family_628ha
        )


    return corrupted_628ha


# ============================================================
# 13. ONE ACTUAL MANIFEST SCENARIO
#
# Use scenario 001 only as a bridge smoke test.
#
# This is NOT an experiment conclusion.
# ============================================================

scenario_smoke_628ha = (
    FD002_CORRUPTION_MANIFEST_628GD
    .iloc[
        0
    ]
)


smoke_id_628ha = str(
    scenario_smoke_628ha[
        "scenario_id"
    ]
)

smoke_family_628ha = str(
    scenario_smoke_628ha[
        "family"
    ]
)

smoke_sensor_628ha = str(
    scenario_smoke_628ha[
        "sensor"
    ]
)


print(
    "\n--- ONE-SCENARIO BRIDGE SMOKE TEST ---"
)

print(
    "Scenario:",
    smoke_id_628ha
)

print(
    "Family  :",
    smoke_family_628ha
)

print(
    "Sensor  :",
    smoke_sensor_628ha
)

print(
    "Severity:",
    scenario_smoke_628ha[
        "severity"
    ]
)


clean_source_snapshot_628ha = (
    dev_raw_628ha[
        model_sensors_628ha
    ]
    .to_numpy(
        dtype=np.float64,
        copy=True
    )
)


corrupted_smoke_raw_628ha = (
    apply_manifest_scenario_628ha(
        dev_raw_628ha,
        scenario_smoke_628ha
    )
)


corrupted_smoke_norm_628ha = (
    normalize_corrupted_sensor_628ha(
        corrupted_smoke_raw_628ha,
        smoke_sensor_628ha
    )
)


X_SMOKE_CORRUPTED_628HA = (
    build_dev_windows_628ha(
        corrupted_smoke_norm_628ha
    )
)


SMOKE_CORRUPTED_PREDICTIONS_628HA = (
    predict_gru_628ha(
        X_SMOKE_CORRUPTED_628HA,
        batch_size=256
    )
)


prediction_delta_628ha = (
    SMOKE_CORRUPTED_PREDICTIONS_628HA
    -
    CLEAN_DEV_PREDICTIONS_REBUILT_628HA
)


print(
    "Corrupted X shape:",
    X_SMOKE_CORRUPTED_628HA.shape
)

print(
    "Predictions       :",
    len(
        SMOKE_CORRUPTED_PREDICTIONS_628HA
    )
)

print(
    "Prediction delta mean abs:",
    float(
        np.mean(
            np.abs(
                prediction_delta_628ha
            )
        )
    )
)

print(
    "Prediction delta max abs :",
    float(
        np.max(
            np.abs(
                prediction_delta_628ha
            )
        )
    )
)


# ============================================================
# 14. CLEAN-SOURCE MUTATION CHECK
# ============================================================

source_after_628ha = (
    dev_raw_628ha[
        model_sensors_628ha
    ]
    .to_numpy(
        dtype=np.float64
    )
)


clean_source_unchanged_628ha = bool(
    np.array_equal(
        clean_source_snapshot_628ha,
        source_after_628ha
    )
)


# ============================================================
# 15. NUMERICAL INTEGRITY
# ============================================================

smoke_X_finite_628ha = bool(
    np.isfinite(
        X_SMOKE_CORRUPTED_628HA
    ).all()
)


smoke_predictions_finite_628ha = bool(
    np.isfinite(
        SMOKE_CORRUPTED_PREDICTIONS_628HA
    ).all()
)


# ============================================================
# 16. FINAL AUDIT
# ============================================================

audit_rows_628ha = [

    {
        "check":
            "fit_raw_normalized_alignment",
        "passed":
            fit_alignment_628ha,
    },

    {
        "check":
            "dev_raw_normalized_alignment",
        "passed":
            dev_alignment_628ha,
    },

    {
        "check":
            "fit_normalization_reproduced",
        "passed":
            (
                best_fit_normalization_error_628ha
                <=
                1e-8
            ),
    },

    {
        "check":
            "dev_normalization_reproduced",
        "passed":
            (
                max_dev_normalization_error_628ha
                <=
                1e-8
            ),
    },

    {
        "check":
            "clean_windows_reproduced",
        "passed":
            (
                clean_window_max_error_628ha
                <=
                1e-7
            ),
    },

    {
        "check":
            "clean_predictions_reproduced",
        "passed":
            (
                clean_prediction_max_error_628ha
                <=
                2e-5
            ),
    },

    {
        "check":
            "corrupted_X_finite",
        "passed":
            smoke_X_finite_628ha,
    },

    {
        "check":
            "corrupted_predictions_finite",
        "passed":
            smoke_predictions_finite_628ha,
    },

    {
        "check":
            "clean_source_unchanged",
        "passed":
            clean_source_unchanged_628ha,
    },
]


FD002_CORRUPTION_MODEL_BRIDGE_AUDIT_628HA = (
    pd.DataFrame(
        audit_rows_628ha
    )
)


print(
    "\n--- FINAL BRIDGE AUDIT ---"
)

display(
    FD002_CORRUPTION_MODEL_BRIDGE_AUDIT_628HA
)


all_bridge_checks_pass_628ha = bool(

    FD002_CORRUPTION_MODEL_BRIDGE_AUDIT_628HA[
        "passed"
    ].all()
)


# ============================================================
# 17. FREEZE BRIDGE CONTRACT IN MEMORY
# ============================================================

FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA = {

    "normalization_ddof":
        int(
            NORMALIZATION_DDOF_628HA
        ),

    "regime_sensor_stats":
        FD002_RECONSTRUCTED_REGIME_SENSOR_STATS_628HA,

    "normalize_corrupted_sensor":
        normalize_corrupted_sensor_628ha,

    "build_dev_windows":
        build_dev_windows_628ha,

    "predict_gru":
        predict_gru_628ha,

    "apply_manifest_scenario":
        apply_manifest_scenario_628ha,

    "deterministic_batch_size":
        256,

    "clean_window_max_error":
        clean_window_max_error_628ha,

    "clean_prediction_max_error":
        clean_prediction_max_error_628ha,

    "all_checks_passed":
        all_bridge_checks_pass_628ha,
}


if not all_bridge_checks_pass_628ha:

    failed_628ha = (

        FD002_CORRUPTION_MODEL_BRIDGE_AUDIT_628HA[
            ~FD002_CORRUPTION_MODEL_BRIDGE_AUDIT_628HA[
                "passed"
            ]
        ][
            "check"
        ]
        .tolist()
    )

    raise RuntimeError(
        "FD002 corruption/model bridge failed: "
        + str(
            failed_628ha
        )
    )


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Normalization ddof resolved :",
    NORMALIZATION_DDOF_628HA
)

print(
    "FIT normalization max error :",
    best_fit_normalization_error_628ha
)

print(
    "DEV normalization max error :",
    max_dev_normalization_error_628ha
)

print(
    "Clean window max error      :",
    clean_window_max_error_628ha
)

print(
    "Clean prediction max error  :",
    clean_prediction_max_error_628ha
)

print(
    "Smoke scenario              :",
    smoke_id_628ha
)

print(
    "Smoke prediction mean |Δ|   :",
    float(
        np.mean(
            np.abs(
                prediction_delta_628ha
            )
        )
    )
)

print(
    "Smoke prediction max |Δ|    :",
    float(
        np.max(
            np.abs(
                prediction_delta_628ha
            )
        )
    )
)

print(
    "All bridge checks passed    :",
    all_bridge_checks_pass_628ha
)

print(
    "Clean source unchanged      :",
    clean_source_unchanged_628ha
)

print(
    "DEV labels used             : False"
)

print(
    "Official TEST labels read   : False"
)

print(
    "TEST scored                 : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Model retrained             : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28H-A ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-B
# FULL FD002 DEV CORRUPTION-RESPONSE EXPERIMENT
#
# PURPOSE
# -------
# Execute every frozen corruption scenario on FD002 DEV:
#
# raw clean DEV
#    -> corruption
#    -> frozen FIT-only normalization
#    -> frozen 30-cycle windows
#    -> frozen GRU
#    -> DEV RUL response metrics
#
# DEV labels ARE used here to quantify corruption damage.
#
# Official TEST labels remain sealed.
# No threshold selection.
# No model retraining.
# No files written.
# ============================================================

import time
import gc
import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-B — FULL FD002 DEV CORRUPTION RESPONSE\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628hb = [

    "FD002_CORRUPTION_MANIFEST_628GD",

    "FD002_DEV_CORRUPTION_BASE_628GC",

    "FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA",

    "CLEAN_DEV_PREDICTIONS_REBUILT_628HA",

    "Y_DEV_PRIMARY_FD002_628D",

    "DEV_WINDOW_KEYS_FD002_628D",
]


print("\n--- REQUIRED OBJECTS ---")

missing_628hb = []

for name_628hb in required_628hb:

    exists_628hb = name_628hb in globals()

    print(
        f"{name_628hb:<48}: "
        f"{exists_628hb}"
    )

    if not exists_628hb:
        missing_628hb.append(name_628hb)


if missing_628hb:

    raise RuntimeError(
        "Missing required objects: "
        + str(missing_628hb)
    )


# ============================================================
# 2. FREEZE LOCAL REFERENCES
# ============================================================

manifest_628hb = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)


dev_raw_628hb = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .copy()
    .reset_index(drop=True)
)


bridge_628hb = (
    FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA
)


apply_scenario_628hb = (
    bridge_628hb[
        "apply_manifest_scenario"
    ]
)


normalize_sensor_628hb = (
    bridge_628hb[
        "normalize_corrupted_sensor"
    ]
)


build_windows_628hb = (
    bridge_628hb[
        "build_dev_windows"
    ]
)


predict_gru_628hb = (
    bridge_628hb[
        "predict_gru"
    ]
)


clean_predictions_628hb = np.asarray(
    CLEAN_DEV_PREDICTIONS_REBUILT_628HA,
    dtype=np.float64
).reshape(-1)


y_dev_628hb = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)


dev_keys_628hb = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)


n_scenarios_628hb = len(
    manifest_628hb
)

n_windows_628hb = len(
    y_dev_628hb
)


print("\n--- EXPERIMENT CONTRACT ---")

print(
    "Frozen scenarios :",
    n_scenarios_628hb
)

print(
    "DEV windows      :",
    n_windows_628hb
)

print(
    "DEV engines      :",
    dev_keys_628hb[
        "unit_id"
    ].nunique()
)

print(
    "Primary target   : RUL_cap_125"
)


if n_scenarios_628hb != 315:

    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )


if len(clean_predictions_628hb) != n_windows_628hb:

    raise RuntimeError(
        "Clean prediction / target length mismatch."
    )


if len(dev_keys_628hb) != n_windows_628hb:

    raise RuntimeError(
        "DEV key / target length mismatch."
    )


if not np.isfinite(
    clean_predictions_628hb
).all():

    raise RuntimeError(
        "Clean predictions contain "
        "non-finite values."
    )


if not np.isfinite(
    y_dev_628hb
).all():

    raise RuntimeError(
        "DEV targets contain "
        "non-finite values."
    )


# ============================================================
# 3. CLEAN SOURCE SNAPSHOT
# ============================================================

model_sensors_628hb = list(
    FD002_MODEL_SENSORS_628GB
)


clean_source_snapshot_628hb = (
    dev_raw_628hb[
        model_sensors_628hb
    ]
    .to_numpy(
        dtype=np.float64,
        copy=True
    )
)


# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def regression_metrics_628hb(
    y_true,
    prediction
):

    y_true = np.asarray(
        y_true,
        dtype=np.float64
    )

    prediction = np.asarray(
        prediction,
        dtype=np.float64
    )


    error = (
        prediction
        -
        y_true
    )


    absolute_error = np.abs(
        error
    )


    return {

        "mae":
            float(
                np.mean(
                    absolute_error
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "mean_error":
            float(
                np.mean(
                    error
                )
            ),

        "median_absolute_error":
            float(
                np.median(
                    absolute_error
                )
            ),

        "p95_absolute_error":
            float(
                np.quantile(
                    absolute_error,
                    0.95
                )
            ),
    }


# ============================================================
# 5. CLEAN BASELINE METRICS
# ============================================================

CLEAN_DEV_METRICS_628HB = (
    regression_metrics_628hb(
        y_dev_628hb,
        clean_predictions_628hb
    )
)


print(
    "\n--- CLEAN DEV BASELINE ---"
)

for key_628hb, value_628hb in (
    CLEAN_DEV_METRICS_628HB.items()
):

    print(
        f"{key_628hb:<24}: "
        f"{value_628hb:.6f}"
    )


clean_mae_628hb = (
    CLEAN_DEV_METRICS_628HB[
        "mae"
    ]
)

clean_rmse_628hb = (
    CLEAN_DEV_METRICS_628HB[
        "rmse"
    ]
)


# ============================================================
# 6. ONE END-TO-END BRIDGE CHECK PER FAMILY
#
# Before 315 runs, confirm every corruption family passes
# through the full bridge.
# ============================================================

print(
    "\n--- FAMILY BRIDGE PREFLIGHT ---"
)


family_bridge_rows_628hb = []


for family_628hb in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    family_row_628hb = (
        manifest_628hb[
            manifest_628hb[
                "family"
            ]
            ==
            family_628hb
        ]
        .iloc[0]
    )


    corrupted_raw_628hb = (
        apply_scenario_628hb(
            dev_raw_628hb,
            family_row_628hb
        )
    )


    corrupted_norm_628hb = (
        normalize_sensor_628hb(
            corrupted_raw_628hb,
            str(
                family_row_628hb[
                    "sensor"
                ]
            )
        )
    )


    X_family_628hb = (
        build_windows_628hb(
            corrupted_norm_628hb
        )
    )


    pred_family_628hb = (
        predict_gru_628hb(
            X_family_628hb,
            batch_size=256
        )
    )


    finite_628hb = bool(
        np.isfinite(
            X_family_628hb
        ).all()
        and
        np.isfinite(
            pred_family_628hb
        ).all()
    )


    family_bridge_rows_628hb.append(
        {
            "family":
                family_628hb,

            "scenario_id":
                str(
                    family_row_628hb[
                        "scenario_id"
                    ]
                ),

            "sensor":
                str(
                    family_row_628hb[
                        "sensor"
                    ]
                ),

            "finite":
                finite_628hb,

            "mean_abs_prediction_shift":
                float(
                    np.mean(
                        np.abs(
                            pred_family_628hb
                            -
                            clean_predictions_628hb
                        )
                    )
                ),
        }
    )


    print(
        f"{family_628hb:<8} "
        f"| finite={finite_628hb} "
        f"| mean |Δpred|="
        f"{family_bridge_rows_628hb[-1]['mean_abs_prediction_shift']:.6f}"
    )


FD002_FAMILY_BRIDGE_PREFLIGHT_628HB = (
    pd.DataFrame(
        family_bridge_rows_628hb
    )
)


if not bool(
    FD002_FAMILY_BRIDGE_PREFLIGHT_628HB[
        "finite"
    ].all()
):

    raise RuntimeError(
        "At least one corruption family "
        "failed the end-to-end bridge."
    )


# Remove temporary preflight arrays.
del corrupted_raw_628hb
del corrupted_norm_628hb
del X_family_628hb
del pred_family_628hb

gc.collect()


# ============================================================
# 7. ALLOCATE FULL PREDICTION MATRIX
#
# Shape:
#   scenarios × DEV windows
#
# float32 keeps memory use modest.
# ============================================================

FD002_CORRUPTED_DEV_PREDICTIONS_628HB = (
    np.empty(
        (
            n_scenarios_628hb,
            n_windows_628hb
        ),
        dtype=np.float32
    )
)


scenario_result_rows_628hb = []


# ============================================================
# 8. FULL 315-SCENARIO EXPERIMENT
# ============================================================

print(
    "\n--- RUNNING 315 FROZEN SCENARIOS ---"
)

experiment_start_628hb = time.time()


for scenario_index_628hb, scenario_row_628hb in (
    manifest_628hb.iterrows()
):

    scenario_start_628hb = time.time()


    scenario_id_628hb = str(
        scenario_row_628hb[
            "scenario_id"
        ]
    )


    family_628hb = str(
        scenario_row_628hb[
            "family"
        ]
    )


    sensor_628hb = str(
        scenario_row_628hb[
            "sensor"
        ]
    )


    severity_628hb = str(
        scenario_row_628hb[
            "severity"
        ]
    )


    # --------------------------------------------------------
    # 8A. Corrupt raw DEV
    # --------------------------------------------------------

    corrupted_raw_628hb = (
        apply_scenario_628hb(
            dev_raw_628hb,
            scenario_row_628hb
        )
    )


    # --------------------------------------------------------
    # 8B. Apply frozen FIT-only normalization
    # --------------------------------------------------------

    corrupted_norm_628hb = (
        normalize_sensor_628hb(
            corrupted_raw_628hb,
            sensor_628hb
        )
    )


    # --------------------------------------------------------
    # 8C. Reconstruct exact DEV windows
    # --------------------------------------------------------

    X_corrupted_628hb = (
        build_windows_628hb(
            corrupted_norm_628hb
        )
    )


    # --------------------------------------------------------
    # 8D. Frozen deterministic GRU inference
    # --------------------------------------------------------

    corrupted_predictions_628hb = (
        predict_gru_628hb(
            X_corrupted_628hb,
            batch_size=256
        )
    )


    if len(
        corrupted_predictions_628hb
    ) != n_windows_628hb:

        raise RuntimeError(
            f"{scenario_id_628hb}: "
            "wrong prediction count."
        )


    if not np.isfinite(
        corrupted_predictions_628hb
    ).all():

        raise RuntimeError(
            f"{scenario_id_628hb}: "
            "non-finite predictions."
        )


    # --------------------------------------------------------
    # 8E. Freeze prediction vector
    # --------------------------------------------------------

    FD002_CORRUPTED_DEV_PREDICTIONS_628HB[
        scenario_index_628hb
    ] = np.asarray(
        corrupted_predictions_628hb,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # 8F. RUL metrics
    # --------------------------------------------------------

    metrics_628hb = (
        regression_metrics_628hb(
            y_dev_628hb,
            corrupted_predictions_628hb
        )
    )


    prediction_shift_628hb = (
        np.asarray(
            corrupted_predictions_628hb,
            dtype=np.float64
        )
        -
        clean_predictions_628hb
    )


    mean_abs_prediction_shift_628hb = float(
        np.mean(
            np.abs(
                prediction_shift_628hb
            )
        )
    )


    max_abs_prediction_shift_628hb = float(
        np.max(
            np.abs(
                prediction_shift_628hb
            )
        )
    )


    mean_prediction_shift_628hb = float(
        np.mean(
            prediction_shift_628hb
        )
    )


    mae_increase_628hb = float(
        metrics_628hb[
            "mae"
        ]
        -
        clean_mae_628hb
    )


    rmse_increase_628hb = float(
        metrics_628hb[
            "rmse"
        ]
        -
        clean_rmse_628hb
    )


    mae_change_pct_628hb = float(
        100.0
        *
        mae_increase_628hb
        /
        clean_mae_628hb
    )


    rmse_change_pct_628hb = float(
        100.0
        *
        rmse_increase_628hb
        /
        clean_rmse_628hb
    )


    scenario_result_rows_628hb.append(
        {

            "scenario_id":
                scenario_id_628hb,

            "scenario_number":
                int(
                    scenario_row_628hb[
                        "scenario_number"
                    ]
                ),

            "family":
                family_628hb,

            "sensor":
                sensor_628hb,

            "severity":
                severity_628hb,

            "alpha":
                scenario_row_628hb[
                    "alpha"
                ],

            "direction":
                scenario_row_628hb[
                    "direction"
                ],

            "requested_fraction":
                scenario_row_628hb[
                    "requested_fraction"
                ],

            "seed":
                scenario_row_628hb[
                    "seed"
                ],

            "mae":
                metrics_628hb[
                    "mae"
                ],

            "rmse":
                metrics_628hb[
                    "rmse"
                ],

            "mean_error":
                metrics_628hb[
                    "mean_error"
                ],

            "median_absolute_error":
                metrics_628hb[
                    "median_absolute_error"
                ],

            "p95_absolute_error":
                metrics_628hb[
                    "p95_absolute_error"
                ],

            "mae_increase":
                mae_increase_628hb,

            "rmse_increase":
                rmse_increase_628hb,

            "mae_change_pct":
                mae_change_pct_628hb,

            "rmse_change_pct":
                rmse_change_pct_628hb,

            "mean_prediction_shift":
                mean_prediction_shift_628hb,

            "mean_abs_prediction_shift":
                mean_abs_prediction_shift_628hb,

            "max_abs_prediction_shift":
                max_abs_prediction_shift_628hb,

            "scenario_runtime_sec":
                float(
                    time.time()
                    -
                    scenario_start_628hb
                ),
        }
    )


    # --------------------------------------------------------
    # 8G. Compact progress reporting
    # --------------------------------------------------------

    completed_628hb = (
        scenario_index_628hb
        +
        1
    )


    if (
        completed_628hb == 1
        or
        completed_628hb % 15 == 0
        or
        completed_628hb == n_scenarios_628hb
    ):

        elapsed_628hb = (
            time.time()
            -
            experiment_start_628hb
        )

        print(
            f"[{completed_628hb:03d}/"
            f"{n_scenarios_628hb}] "
            f"{scenario_id_628hb} "
            f"| {family_628hb:<7} "
            f"| {sensor_628hb:<9} "
            f"| {severity_628hb:<6} "
            f"| MAE={metrics_628hb['mae']:.4f} "
            f"| ΔMAE={mae_increase_628hb:+.4f} "
            f"| elapsed={elapsed_628hb:.1f}s"
        )


    # Release large temporary objects.
    del corrupted_raw_628hb
    del corrupted_norm_628hb
    del X_corrupted_628hb
    del corrupted_predictions_628hb
    del prediction_shift_628hb


    if completed_628hb % 25 == 0:
        gc.collect()


experiment_runtime_628hb = float(
    time.time()
    -
    experiment_start_628hb
)


# ============================================================
# 9. BUILD SCENARIO-LEVEL RESULT TABLE
# ============================================================

FD002_DEV_CORRUPTION_RESPONSE_628HB = (
    pd.DataFrame(
        scenario_result_rows_628hb
    )
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)


print(
    "\n--- FULL RESPONSE TABLE CREATED ---"
)

print(
    "Rows:",
    len(
        FD002_DEV_CORRUPTION_RESPONSE_628HB
    )
)

print(
    "Columns:",
    len(
        FD002_DEV_CORRUPTION_RESPONSE_628HB.columns
    )
)

print(
    "Prediction matrix:",
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB.shape
)

print(
    "Total runtime:",
    experiment_runtime_628hb,
    "seconds"
)


# ============================================================
# 10. FINAL INTEGRITY AUDIT
# ============================================================

clean_source_after_628hb = (
    dev_raw_628hb[
        model_sensors_628hb
    ]
    .to_numpy(
        dtype=np.float64
    )
)


clean_source_unchanged_628hb = bool(
    np.array_equal(
        clean_source_snapshot_628hb,
        clean_source_after_628hb
    )
)


scenario_ids_unique_628hb = bool(
    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        "scenario_id"
    ].nunique()
    ==
    315
)


scenario_order_exact_628hb = bool(
    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        "scenario_id"
    ].tolist()
    ==
    manifest_628hb[
        "scenario_id"
    ].tolist()
)


response_numeric_cols_628hb = [

    "mae",
    "rmse",
    "mean_error",
    "median_absolute_error",
    "p95_absolute_error",
    "mae_increase",
    "rmse_increase",
    "mae_change_pct",
    "rmse_change_pct",
    "mean_prediction_shift",
    "mean_abs_prediction_shift",
    "max_abs_prediction_shift",
]


summary_numeric_finite_628hb = bool(
    np.isfinite(
        FD002_DEV_CORRUPTION_RESPONSE_628HB[
            response_numeric_cols_628hb
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)


prediction_matrix_finite_628hb = bool(
    np.isfinite(
        FD002_CORRUPTED_DEV_PREDICTIONS_628HB
    ).all()
)


audit_rows_628hb = [

    {
        "check":
            "scenario_count_315",

        "passed":
            (
                len(
                    FD002_DEV_CORRUPTION_RESPONSE_628HB
                )
                ==
                315
            ),
    },

    {
        "check":
            "scenario_ids_unique",

        "passed":
            scenario_ids_unique_628hb,
    },

    {
        "check":
            "scenario_order_exact",

        "passed":
            scenario_order_exact_628hb,
    },

    {
        "check":
            "prediction_matrix_shape",

        "passed":
            (
                FD002_CORRUPTED_DEV_PREDICTIONS_628HB.shape
                ==
                (
                    315,
                    n_windows_628hb
                )
            ),
    },

    {
        "check":
            "prediction_matrix_finite",

        "passed":
            prediction_matrix_finite_628hb,
    },

    {
        "check":
            "summary_numeric_finite",

        "passed":
            summary_numeric_finite_628hb,
    },

    {
        "check":
            "clean_source_unchanged",

        "passed":
            clean_source_unchanged_628hb,
    },
]


FD002_DEV_CORRUPTION_RESPONSE_AUDIT_628HB = (
    pd.DataFrame(
        audit_rows_628hb
    )
)


print(
    "\n--- FINAL INTEGRITY AUDIT ---"
)

display(
    FD002_DEV_CORRUPTION_RESPONSE_AUDIT_628HB
)


all_response_checks_628hb = bool(

    FD002_DEV_CORRUPTION_RESPONSE_AUDIT_628HB[
        "passed"
    ].all()
)


if not all_response_checks_628hb:

    failed_628hb = (

        FD002_DEV_CORRUPTION_RESPONSE_AUDIT_628HB[
            ~FD002_DEV_CORRUPTION_RESPONSE_AUDIT_628HB[
                "passed"
            ]
        ][
            "check"
        ]
        .tolist()
    )

    raise RuntimeError(
        "FD002 corruption response "
        "experiment failed audit: "
        +
        str(
            failed_628hb
        )
    )


# ============================================================
# 11. FAMILY × SEVERITY SUMMARY
# ============================================================

FD002_FAMILY_SEVERITY_RESPONSE_628HB = (

    FD002_DEV_CORRUPTION_RESPONSE_628HB

    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False
    )

    .agg(

        scenarios=(
            "scenario_id",
            "count"
        ),

        median_mae_increase=(
            "mae_increase",
            "median"
        ),

        mean_mae_increase=(
            "mae_increase",
            "mean"
        ),

        max_mae_increase=(
            "mae_increase",
            "max"
        ),

        median_rmse_increase=(
            "rmse_increase",
            "median"
        ),

        mean_abs_prediction_shift=(
            "mean_abs_prediction_shift",
            "mean"
        ),

        max_prediction_shift=(
            "max_abs_prediction_shift",
            "max"
        ),
    )
)


severity_order_628hb = pd.CategoricalDtype(
    categories=[
        "low",
        "medium",
        "high",
    ],
    ordered=True
)


FD002_FAMILY_SEVERITY_RESPONSE_628HB[
    "severity"
] = (
    FD002_FAMILY_SEVERITY_RESPONSE_628HB[
        "severity"
    ]
    .astype(
        severity_order_628hb
    )
)


FD002_FAMILY_SEVERITY_RESPONSE_628HB = (

    FD002_FAMILY_SEVERITY_RESPONSE_628HB

    .sort_values(
        [
            "family",
            "severity",
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n--- FAMILY × SEVERITY RESPONSE ---"
)

display(
    FD002_FAMILY_SEVERITY_RESPONSE_628HB
)


# ============================================================
# 12. MOST DAMAGING SCENARIOS
# ============================================================

print(
    "\n--- TOP 15 SCENARIOS BY MAE INCREASE ---"
)

display(

    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "mae",
            "mae_increase",
            "mae_change_pct",
            "rmse",
            "rmse_increase",
            "mean_abs_prediction_shift",
            "max_abs_prediction_shift",
        ]
    ]

    .sort_values(
        "mae_increase",
        ascending=False
    )

    .head(15)

    .reset_index(
        drop=True
    )
)


# ============================================================
# 13. LEAST DAMAGING / APPARENTLY BENEFICIAL SCENARIOS
#
# Corruption can occasionally reduce DEV error by chance.
# We retain those outcomes rather than hiding them.
# ============================================================

print(
    "\n--- BOTTOM 15 SCENARIOS BY MAE INCREASE ---"
)

display(

    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "mae",
            "mae_increase",
            "mae_change_pct",
            "rmse",
            "rmse_increase",
            "mean_abs_prediction_shift",
        ]
    ]

    .sort_values(
        "mae_increase",
        ascending=True
    )

    .head(15)

    .reset_index(
        drop=True
    )
)


# ============================================================
# 14. GLOBAL RESPONSE SUMMARY
# ============================================================

harmful_mae_count_628hb = int(
    (
        FD002_DEV_CORRUPTION_RESPONSE_628HB[
            "mae_increase"
        ]
        >
        0.0
    ).sum()
)


harmful_rmse_count_628hb = int(
    (
        FD002_DEV_CORRUPTION_RESPONSE_628HB[
            "rmse_increase"
        ]
        >
        0.0
    ).sum()
)


median_mae_increase_628hb = float(
    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        "mae_increase"
    ].median()
)


max_mae_increase_628hb = float(
    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        "mae_increase"
    ].max()
)


median_prediction_shift_628hb = float(
    FD002_DEV_CORRUPTION_RESPONSE_628HB[
        "mean_abs_prediction_shift"
    ].median()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Frozen scenarios executed :",
    len(
        FD002_DEV_CORRUPTION_RESPONSE_628HB
    )
)

print(
    "Prediction matrix shape   :",
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB.shape
)

print(
    "Clean DEV MAE             :",
    clean_mae_628hb
)

print(
    "Clean DEV RMSE            :",
    clean_rmse_628hb
)

print(
    "Scenarios with MAE > clean:",
    harmful_mae_count_628hb,
    "/ 315"
)

print(
    "Scenarios RMSE > clean    :",
    harmful_rmse_count_628hb,
    "/ 315"
)

print(
    "Median MAE increase       :",
    median_mae_increase_628hb
)

print(
    "Maximum MAE increase      :",
    max_mae_increase_628hb
)

print(
    "Median mean |prediction Δ|:",
    median_prediction_shift_628hb
)

print(
    "Experiment runtime (sec)  :",
    experiment_runtime_628hb
)

print(
    "All response checks passed:",
    all_response_checks_628hb
)

print(
    "Clean DEV source unchanged:",
    clean_source_unchanged_628hb
)

print(
    "DEV labels used           : True"
)

print(
    "Official TEST labels read : False"
)

print(
    "TEST scored               : False"
)

print(
    "Threshold selected        : False"
)

print(
    "Model retrained           : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.28H-B ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C0
# CORRUPTED RELIABILITY / UNCERTAINTY BRIDGE PRE-FLIGHT
#
# PURPOSE
# -------
# Before running MC-dropout for all 315 corruption scenarios,
# verify that we can exactly/near-exactly reproduce:
#
#   raw MC std
#       -> phase-adjusted uncertainty
#       -> G1 uncertainty risk
#       -> corrected engine-balanced G2 risk
#
# Then test two representative corrupted scenarios.
#
# DEV only.
# Official TEST labels remain sealed.
# No threshold selection.
# No model retraining.
# No files written.
# ============================================================

import time
import gc
import numpy as np
import pandas as pd
import torch


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0 — CORRUPTED RELIABILITY BRIDGE PRE-FLIGHT\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628hc0 = [

    # Frozen model / sequence objects
    "MODEL_FD002_628EB",
    "X_DEV_FD002_628D",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_WINDOW_KEYS_FD002_628D",

    # Clean deterministic / MC references
    "CLEAN_DEV_PREDICTIONS_REBUILT_628HA",
    "DEV_MC_STD_FD002_628FAR2",

    # Frozen clean reliability chain
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",

    # Corrected G2 reference
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",

    # Corruption experiment objects
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "FD002_DEV_CORRUPTION_RESPONSE_628HB",
]


print("\n--- REQUIRED OBJECTS ---")

missing_628hc0 = []

for name_628hc0 in required_628hc0:

    exists_628hc0 = name_628hc0 in globals()

    print(
        f"{name_628hc0:<48}: "
        f"{exists_628hc0}"
    )

    if not exists_628hc0:
        missing_628hc0.append(
            name_628hc0
        )


if missing_628hc0:

    raise RuntimeError(
        "Missing required objects: "
        + str(missing_628hc0)
    )


# ============================================================
# 2. LOCAL FROZEN REFERENCES
# ============================================================

model_628hc0 = MODEL_FD002_628EB

device_628hc0 = next(
    model_628hc0.parameters()
).device


X_dev_628hc0 = np.asarray(
    X_DEV_FD002_628D,
    dtype=np.float32
)


y_dev_628hc0 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)


clean_pred_628hc0 = np.asarray(
    CLEAN_DEV_PREDICTIONS_REBUILT_628HA,
    dtype=np.float64
).reshape(-1)


clean_mc_std_stored_628hc0 = np.asarray(
    DEV_MC_STD_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)


clean_adjusted_stored_628hc0 = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


clean_g1_stored_628hc0 = np.asarray(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)


clean_g2_stored_628hc0 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


g2_reference_values_628hc0 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


g2_reference_weights_628hc0 = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


manifest_628hc0 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)


dev_raw_628hc0 = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .copy()
    .reset_index(drop=True)
)


bridge_628hc0 = (
    FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA
)


apply_scenario_628hc0 = (
    bridge_628hc0[
        "apply_manifest_scenario"
    ]
)


normalize_sensor_628hc0 = (
    bridge_628hc0[
        "normalize_corrupted_sensor"
    ]
)


build_windows_628hc0 = (
    bridge_628hc0[
        "build_dev_windows"
    ]
)


predict_gru_628hc0 = (
    bridge_628hc0[
        "predict_gru"
    ]
)


print("\n--- FROZEN CONTRACT ---")

print(
    "Device              :",
    device_628hc0
)

print(
    "DEV windows         :",
    len(
        X_dev_628hc0
    )
)

print(
    "MC passes           : 50"
)

print(
    "MC batch size       : 512"
)

print(
    "MC seed             : 42"
)

print(
    "MC STD convention   : ddof=0"
)

print(
    "Phase recipe        : det_pred_pct_b5"
)

print(
    "G1 reference        : clean DEV windows"
)

print(
    "G2 reference values :",
    len(
        g2_reference_values_628hc0
    )
)

print(
    "G2 weight sum       :",
    float(
        g2_reference_weights_628hc0.sum()
    )
)


# ============================================================
# 3. MC-DROPOUT FUNCTION
#
# Fixed seed = common random numbers across scenarios.
# This reduces Monte-Carlo noise when comparing clean vs corrupt.
# ============================================================

def mc_dropout_predict_628hc0(
    X,
    model,
    passes=50,
    batch_size=512,
    seed=42,
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    n = len(X)

    all_predictions = np.empty(
        (
            passes,
            n
        ),
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Freeze RNG convention
    # --------------------------------------------------------

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    original_training_state = (
        model.training
    )


    # Enable dropout.
    model.train()


    start_time = time.time()


    with torch.no_grad():

        for pass_idx in range(
            passes
        ):

            for start_idx in range(
                0,
                n,
                batch_size
            ):

                end_idx = min(
                    start_idx
                    +
                    batch_size,
                    n
                )


                xb = torch.as_tensor(
                    X[
                        start_idx:end_idx
                    ],
                    dtype=torch.float32,
                    device=device_628hc0
                )


                pred = (
                    model(
                        xb
                    )
                    .reshape(-1)
                    .detach()
                    .cpu()
                    .numpy()
                )


                all_predictions[
                    pass_idx,
                    start_idx:end_idx
                ] = pred


    # Restore original model state.
    if original_training_state:
        model.train()
    else:
        model.eval()


    mc_mean = (
        all_predictions
        .mean(
            axis=0
        )
        .astype(
            np.float64
        )
    )


    mc_std = (
        all_predictions
        .std(
            axis=0,
            ddof=0
        )
        .astype(
            np.float64
        )
    )


    runtime = float(
        time.time()
        -
        start_time
    )


    return (
        mc_mean,
        mc_std,
        runtime
    )


# ============================================================
# 4. CLEAN MC-DROPOUT REPRODUCTION
# ============================================================

print(
    "\n--- CLEAN MC-DROPOUT REPRODUCTION ---"
)

(
    CLEAN_MC_MEAN_RECHECK_628HC0,
    CLEAN_MC_STD_RECHECK_628HC0,
    clean_mc_runtime_628hc0

) = mc_dropout_predict_628hc0(

    X_dev_628hc0,
    model_628hc0,

    passes=50,
    batch_size=512,
    seed=42,
)


clean_mc_max_error_628hc0 = float(
    np.max(
        np.abs(
            CLEAN_MC_STD_RECHECK_628HC0
            -
            clean_mc_std_stored_628hc0
        )
    )
)


clean_mc_mean_error_628hc0 = float(
    np.mean(
        np.abs(
            CLEAN_MC_STD_RECHECK_628HC0
            -
            clean_mc_std_stored_628hc0
        )
    )
)


print(
    "Stored MC std count       :",
    len(
        clean_mc_std_stored_628hc0
    )
)

print(
    "Reproduced MC std count   :",
    len(
        CLEAN_MC_STD_RECHECK_628HC0
    )
)

print(
    "Maximum absolute std error:",
    clean_mc_max_error_628hc0
)

print(
    "Mean absolute std error   :",
    clean_mc_mean_error_628hc0
)

print(
    "Runtime (sec)             :",
    clean_mc_runtime_628hc0
)


# We allow only tiny floating-point differences.
clean_mc_reproduced_628hc0 = bool(

    np.allclose(
        CLEAN_MC_STD_RECHECK_628HC0,
        clean_mc_std_stored_628hc0,
        atol=5e-5,
        rtol=1e-6
    )
)


print(
    "MC uncertainty reproduced :",
    clean_mc_reproduced_628hc0
)


if not clean_mc_reproduced_628hc0:

    raise RuntimeError(
        "Stored clean MC-dropout uncertainty "
        "was not reproduced. Stop before "
        "corrupted uncertainty generation."
    )


# ============================================================
# 5. RECONSTRUCT FROZEN 5-BIN PHASE REFERENCE
#
# det_pred_pct_b5
# ============================================================

phase_edges_628hc0 = np.quantile(
    clean_pred_628hc0,
    [
        0.20,
        0.40,
        0.60,
        0.80,
    ]
)


def assign_phase_628hc0(
    deterministic_prediction
):

    values = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    )

    return np.digitize(
        values,
        bins=phase_edges_628hc0,
        right=False
    )


clean_phase_628hc0 = (
    assign_phase_628hc0(
        clean_pred_628hc0
    )
)


phase_counts_628hc0 = np.bincount(
    clean_phase_628hc0,
    minlength=5
)


print(
    "\n--- RECONSTRUCTED PHASE REFERENCE ---"
)

print(
    "Interior edges:",
    phase_edges_628hc0.tolist()
)

print(
    "Phase counts  :",
    phase_counts_628hc0.tolist()
)


# ============================================================
# 6. EMPIRICAL MID-RANK CDF
# ============================================================

def empirical_midrank_cdf_628hc0(
    x,
    reference
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    reference = np.sort(
        np.asarray(
            reference,
            dtype=np.float64
        )
    )


    n_ref = len(
        reference
    )


    left = np.searchsorted(
        reference,
        x,
        side="left"
    )


    right = np.searchsorted(
        reference,
        x,
        side="right"
    )


    result = (
        left
        +
        0.5
        *
        (
            right
            -
            left
        )
    ) / float(
        n_ref
    )


    return np.clip(
        result,
        0.0,
        1.0
    )


# ============================================================
# 7. PHASE-ADJUSTMENT FUNCTION
# ============================================================

phase_mc_reference_628hc0 = {

    phase_id:
        clean_mc_std_stored_628hc0[
            clean_phase_628hc0
            ==
            phase_id
        ]

    for phase_id in range(5)
}


def phase_adjust_uncertainty_628hc0(
    deterministic_prediction,
    mc_std_values
):

    deterministic_prediction = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    )

    mc_std_values = np.asarray(
        mc_std_values,
        dtype=np.float64
    )


    phase_id = assign_phase_628hc0(
        deterministic_prediction
    )


    adjusted = np.empty(
        len(
            mc_std_values
        ),
        dtype=np.float64
    )


    for p in range(5):

        mask = (
            phase_id
            ==
            p
        )


        if not np.any(
            mask
        ):
            continue


        adjusted[
            mask
        ] = empirical_midrank_cdf_628hc0(

            mc_std_values[
                mask
            ],

            phase_mc_reference_628hc0[
                p
            ],
        )


    return adjusted


# ============================================================
# 8. CLEAN PHASE-ADJUSTMENT REPRODUCTION
# ============================================================

clean_adjusted_rebuilt_628hc0 = (
    phase_adjust_uncertainty_628hc0(

        clean_pred_628hc0,

        clean_mc_std_stored_628hc0
    )
)


clean_adjusted_max_error_628hc0 = float(
    np.max(
        np.abs(
            clean_adjusted_rebuilt_628hc0
            -
            clean_adjusted_stored_628hc0
        )
    )
)


print(
    "\n--- PHASE-ADJUSTMENT REPRODUCTION ---"
)

print(
    "Maximum absolute error:",
    clean_adjusted_max_error_628hc0
)


phase_adjustment_exact_628hc0 = bool(

    np.allclose(
        clean_adjusted_rebuilt_628hc0,
        clean_adjusted_stored_628hc0,
        atol=1e-12,
        rtol=0.0
    )
)


print(
    "Phase adjustment exact:",
    phase_adjustment_exact_628hc0
)


if not phase_adjustment_exact_628hc0:

    raise RuntimeError(
        "Frozen phase-adjusted uncertainty "
        "was not exactly reproduced."
    )


# ============================================================
# 9. G1 = RIGHT EMPIRICAL CDF ON CLEAN DEV REFERENCE
# ============================================================

g1_reference_628hc0 = np.sort(
    clean_adjusted_stored_628hc0
)


def right_ecdf_628hc0(
    x,
    reference
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    reference = np.asarray(
        reference,
        dtype=np.float64
    )


    return (
        np.searchsorted(
            reference,
            x,
            side="right"
        )
        /
        float(
            len(
                reference
            )
        )
    )


clean_g1_rebuilt_628hc0 = (
    right_ecdf_628hc0(

        clean_adjusted_stored_628hc0,

        g1_reference_628hc0
    )
)


clean_g1_max_error_628hc0 = float(
    np.max(
        np.abs(
            clean_g1_rebuilt_628hc0
            -
            clean_g1_stored_628hc0
        )
    )
)


g1_exact_628hc0 = bool(

    np.allclose(
        clean_g1_rebuilt_628hc0,
        clean_g1_stored_628hc0,
        atol=1e-12,
        rtol=0.0
    )
)


print(
    "\n--- G1 REPRODUCTION ---"
)

print(
    "Maximum absolute error:",
    clean_g1_max_error_628hc0
)

print(
    "G1 exact              :",
    g1_exact_628hc0
)


if not g1_exact_628hc0:

    raise RuntimeError(
        "Frozen G1 uncertainty risk "
        "was not exactly reproduced."
    )


# ============================================================
# 10. ENGINE-BALANCED WEIGHTED MID-RANK G2
# ============================================================

g2_sort_order_628hc0 = np.argsort(
    g2_reference_values_628hc0,
    kind="mergesort"
)


g2_values_sorted_628hc0 = (
    g2_reference_values_628hc0[
        g2_sort_order_628hc0
    ]
)


g2_weights_sorted_628hc0 = (
    g2_reference_weights_628hc0[
        g2_sort_order_628hc0
    ]
)


g2_weight_prefix_628hc0 = np.concatenate(
    [
        np.array(
            [0.0],
            dtype=np.float64
        ),

        np.cumsum(
            g2_weights_sorted_628hc0
        ),
    ]
)


def weighted_midrank_cdf_628hc0(
    x
):

    x = np.asarray(
        x,
        dtype=np.float64
    )


    left = np.searchsorted(
        g2_values_sorted_628hc0,
        x,
        side="left"
    )


    right = np.searchsorted(
        g2_values_sorted_628hc0,
        x,
        side="right"
    )


    mass_less = (
        g2_weight_prefix_628hc0[
            left
        ]
    )


    mass_equal = (
        g2_weight_prefix_628hc0[
            right
        ]
        -
        g2_weight_prefix_628hc0[
            left
        ]
    )


    result = (
        mass_less
        +
        0.5
        *
        mass_equal
    )


    return np.clip(
        result,
        0.0,
        1.0
    )


clean_g2_rebuilt_628hc0 = (
    weighted_midrank_cdf_628hc0(
        clean_g1_stored_628hc0
    )
)


clean_g2_max_error_628hc0 = float(
    np.max(
        np.abs(
            clean_g2_rebuilt_628hc0
            -
            clean_g2_stored_628hc0
        )
    )
)


g2_exact_628hc0 = bool(

    np.allclose(
        clean_g2_rebuilt_628hc0,
        clean_g2_stored_628hc0,
        atol=1e-12,
        rtol=0.0
    )
)


print(
    "\n--- CORRECTED G2 REPRODUCTION ---"
)

print(
    "Maximum absolute error:",
    clean_g2_max_error_628hc0
)

print(
    "Corrected G2 exact    :",
    g2_exact_628hc0
)


if not g2_exact_628hc0:

    raise RuntimeError(
        "Frozen corrected G2 risk "
        "was not exactly reproduced."
    )


# ============================================================
# 11. COMPLETE FROZEN RELIABILITY TRANSFORM
# ============================================================

def reliability_chain_628hc0(
    deterministic_prediction,
    mc_std_values
):

    adjusted = (
        phase_adjust_uncertainty_628hc0(

            deterministic_prediction,

            mc_std_values
        )
    )


    g1 = (
        right_ecdf_628hc0(
            adjusted,
            g1_reference_628hc0
        )
    )


    g2 = (
        weighted_midrank_cdf_628hc0(
            g1
        )
    )


    return (
        adjusted,
        g1,
        g2
    )


# ============================================================
# 12. SELECT REPRESENTATIVE CORRUPTION SCENARIOS
# ============================================================

worst_row_628hc0 = (

    FD002_DEV_CORRUPTION_RESPONSE_628HB

    .sort_values(
        "mae_increase",
        ascending=False
    )

    .iloc[0]
)


scenario_ids_628hc0 = [

    "FD002_SCENARIO_001",

    str(
        worst_row_628hc0[
            "scenario_id"
        ]
    ),
]


# Remove duplicate if somehow identical.
scenario_ids_628hc0 = list(
    dict.fromkeys(
        scenario_ids_628hc0
    )
)


print(
    "\n--- REPRESENTATIVE SCENARIOS ---"
)

print(
    scenario_ids_628hc0
)


# ============================================================
# 13. CORRUPTED RELIABILITY PRE-FLIGHT
# ============================================================

scenario_reliability_rows_628hc0 = []


for scenario_id_628hc0 in (
    scenario_ids_628hc0
):

    scenario_row_628hc0 = (

        manifest_628hc0[
            manifest_628hc0[
                "scenario_id"
            ]
            ==
            scenario_id_628hc0
        ]

        .iloc[0]
    )


    scenario_index_628hc0 = int(
        scenario_row_628hc0.name
    )


    family_628hc0 = str(
        scenario_row_628hc0[
            "family"
        ]
    )


    sensor_628hc0 = str(
        scenario_row_628hc0[
            "sensor"
        ]
    )


    severity_628hc0 = str(
        scenario_row_628hc0[
            "severity"
        ]
    )


    print(
        "\n"
        "------------------------------------------------------------"
    )

    print(
        "Scenario :",
        scenario_id_628hc0
    )

    print(
        "Family   :",
        family_628hc0
    )

    print(
        "Sensor   :",
        sensor_628hc0
    )

    print(
        "Severity :",
        severity_628hc0
    )


    # --------------------------------------------------------
    # Corrupt raw DEV
    # --------------------------------------------------------

    corrupted_raw_628hc0 = (
        apply_scenario_628hc0(
            dev_raw_628hc0,
            scenario_row_628hc0
        )
    )


    # --------------------------------------------------------
    # Frozen FIT-only normalization
    # --------------------------------------------------------

    corrupted_norm_628hc0 = (
        normalize_sensor_628hc0(
            corrupted_raw_628hc0,
            sensor_628hc0
        )
    )


    # --------------------------------------------------------
    # Exact DEV windows
    # --------------------------------------------------------

    X_corrupted_628hc0 = (
        build_windows_628hc0(
            corrupted_norm_628hc0
        )
    )


    # --------------------------------------------------------
    # Deterministic prediction
    # --------------------------------------------------------

    deterministic_corrupt_628hc0 = (
        predict_gru_628hc0(
            X_corrupted_628hc0,
            batch_size=256
        )
    )


    deterministic_corrupt_628hc0 = np.asarray(
        deterministic_corrupt_628hc0,
        dtype=np.float64
    )


    # --------------------------------------------------------
    # Check deterministic prediction against H-B matrix
    # --------------------------------------------------------

    frozen_hb_prediction_628hc0 = np.asarray(
        FD002_CORRUPTED_DEV_PREDICTIONS_628HB[
            scenario_index_628hc0
        ],
        dtype=np.float64
    )


    deterministic_bridge_error_628hc0 = float(
        np.max(
            np.abs(
                deterministic_corrupt_628hc0
                -
                frozen_hb_prediction_628hc0
            )
        )
    )


    print(
        "Deterministic H-B max error:",
        deterministic_bridge_error_628hc0
    )


    if deterministic_bridge_error_628hc0 > 5e-5:

        raise RuntimeError(
            scenario_id_628hc0
            +
            ": deterministic bridge "
            "does not reproduce H-B."
        )


    # --------------------------------------------------------
    # MC dropout under corruption
    # --------------------------------------------------------

    (
        mc_mean_corrupt_628hc0,
        mc_std_corrupt_628hc0,
        scenario_mc_runtime_628hc0

    ) = mc_dropout_predict_628hc0(

        X_corrupted_628hc0,
        model_628hc0,

        passes=50,
        batch_size=512,
        seed=42,
    )


    # --------------------------------------------------------
    # Frozen reliability chain
    # --------------------------------------------------------

    (
        adjusted_corrupt_628hc0,
        g1_corrupt_628hc0,
        g2_corrupt_628hc0

    ) = reliability_chain_628hc0(

        deterministic_corrupt_628hc0,

        mc_std_corrupt_628hc0
    )


    # --------------------------------------------------------
    # DEV corruption damage
    # --------------------------------------------------------

    corrupt_abs_error_628hc0 = np.abs(
        deterministic_corrupt_628hc0
        -
        y_dev_628hc0
    )


    clean_abs_error_628hc0 = np.abs(
        clean_pred_628hc0
        -
        y_dev_628hc0
    )


    error_increase_628hc0 = (
        corrupt_abs_error_628hc0
        -
        clean_abs_error_628hc0
    )


    scenario_reliability_rows_628hc0.append(
        {

            "scenario_id":
                scenario_id_628hc0,

            "family":
                family_628hc0,

            "sensor":
                sensor_628hc0,

            "severity":
                severity_628hc0,

            "mc_runtime_sec":
                scenario_mc_runtime_628hc0,

            "mean_mc_std":
                float(
                    np.mean(
                        mc_std_corrupt_628hc0
                    )
                ),

            "median_mc_std":
                float(
                    np.median(
                        mc_std_corrupt_628hc0
                    )
                ),

            "median_adjusted_uncertainty":
                float(
                    np.median(
                        adjusted_corrupt_628hc0
                    )
                ),

            "mean_adjusted_uncertainty":
                float(
                    np.mean(
                        adjusted_corrupt_628hc0
                    )
                ),

            "median_g1_risk":
                float(
                    np.median(
                        g1_corrupt_628hc0
                    )
                ),

            "mean_g1_risk":
                float(
                    np.mean(
                        g1_corrupt_628hc0
                    )
                ),

            "median_final_g2_risk":
                float(
                    np.median(
                        g2_corrupt_628hc0
                    )
                ),

            "mean_final_g2_risk":
                float(
                    np.mean(
                        g2_corrupt_628hc0
                    )
                ),

            "max_final_g2_risk":
                float(
                    np.max(
                        g2_corrupt_628hc0
                    )
                ),

            "mean_abs_error":
                float(
                    np.mean(
                        corrupt_abs_error_628hc0
                    )
                ),

            "mean_error_increase":
                float(
                    np.mean(
                        error_increase_628hc0
                    )
                ),

            "mean_abs_prediction_shift":
                float(
                    np.mean(
                        np.abs(
                            deterministic_corrupt_628hc0
                            -
                            clean_pred_628hc0
                        )
                    )
                ),

            "finite":
                bool(
                    np.isfinite(
                        mc_std_corrupt_628hc0
                    ).all()
                    and
                    np.isfinite(
                        adjusted_corrupt_628hc0
                    ).all()
                    and
                    np.isfinite(
                        g1_corrupt_628hc0
                    ).all()
                    and
                    np.isfinite(
                        g2_corrupt_628hc0
                    ).all()
                ),
        }
    )


    print(
        "MC runtime (sec)          :",
        scenario_mc_runtime_628hc0
    )

    print(
        "Mean raw MC std           :",
        float(
            np.mean(
                mc_std_corrupt_628hc0
            )
        )
    )

    print(
        "Median adjusted-U         :",
        float(
            np.median(
                adjusted_corrupt_628hc0
            )
        )
    )

    print(
        "Median final G2 risk      :",
        float(
            np.median(
                g2_corrupt_628hc0
            )
        )
    )

    print(
        "Mean final G2 risk        :",
        float(
            np.mean(
                g2_corrupt_628hc0
            )
        )
    )

    print(
        "Mean corruption Δerror    :",
        float(
            np.mean(
                error_increase_628hc0
            )
        )
    )


    del corrupted_raw_628hc0
    del corrupted_norm_628hc0
    del X_corrupted_628hc0

    gc.collect()


FD002_CORRUPTED_RELIABILITY_PREFLIGHT_628HC0 = (
    pd.DataFrame(
        scenario_reliability_rows_628hc0
    )
)


print(
    "\n--- REPRESENTATIVE CORRUPTED RELIABILITY RESULTS ---"
)

display(
    FD002_CORRUPTED_RELIABILITY_PREFLIGHT_628HC0
)


# ============================================================
# 14. RUNTIME PROJECTION
# ============================================================

mean_scenario_runtime_628hc0 = float(

    FD002_CORRUPTED_RELIABILITY_PREFLIGHT_628HC0[
        "mc_runtime_sec"
    ].mean()
)


projected_full_seconds_628hc0 = float(
    315
    *
    mean_scenario_runtime_628hc0
)


print(
    "\n--- FULL-RUN RUNTIME PROJECTION ---"
)

print(
    "Mean MC runtime/scenario :",
    mean_scenario_runtime_628hc0,
    "seconds"
)

print(
    "Projected 315 scenarios  :",
    projected_full_seconds_628hc0,
    "seconds"
)

print(
    "Projected minutes        :",
    projected_full_seconds_628hc0
    /
    60.0
)


# ============================================================
# 15. FINAL AUDIT
# ============================================================

all_corrupt_finite_628hc0 = bool(

    FD002_CORRUPTED_RELIABILITY_PREFLIGHT_628HC0[
        "finite"
    ].all()
)


final_checks_628hc0 = pd.DataFrame(

    [

        {
            "check":
                "clean_mc_reproduced",

            "passed":
                clean_mc_reproduced_628hc0,
        },

        {
            "check":
                "phase_adjustment_exact",

            "passed":
                phase_adjustment_exact_628hc0,
        },

        {
            "check":
                "g1_exact",

            "passed":
                g1_exact_628hc0,
        },

        {
            "check":
                "g2_exact",

            "passed":
                g2_exact_628hc0,
        },

        {
            "check":
                "representative_corruptions_finite",

            "passed":
                all_corrupt_finite_628hc0,
        },
    ]
)


print(
    "\n--- FINAL H-C0 AUDIT ---"
)

display(
    final_checks_628hc0
)


all_hc0_checks_passed_628hc0 = bool(
    final_checks_628hc0[
        "passed"
    ].all()
)


if not all_hc0_checks_passed_628hc0:

    failed_checks_628hc0 = (

        final_checks_628hc0[
            ~final_checks_628hc0[
                "passed"
            ]
        ][
            "check"
        ]
        .tolist()
    )

    raise RuntimeError(
        "H-C0 pre-flight failed: "
        +
        str(
            failed_checks_628hc0
        )
    )


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Clean MC reproduced        :",
    clean_mc_reproduced_628hc0
)

print(
    "Phase adjustment exact     :",
    phase_adjustment_exact_628hc0
)

print(
    "G1 calibration exact       :",
    g1_exact_628hc0
)

print(
    "Corrected G2 exact         :",
    g2_exact_628hc0
)

print(
    "Corrupted examples finite  :",
    all_corrupt_finite_628hc0
)

print(
    "Representative scenarios   :",
    scenario_ids_628hc0
)

print(
    "Projected full runtime min :",
    projected_full_seconds_628hc0
    /
    60.0
)

print(
    "DEV labels used            : True"
)

print(
    "Official TEST labels read  : False"
)

print(
    "TEST scored                : False"
)

print(
    "Threshold selected         : False"
)

print(
    "Model retrained            : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.28H-C0 ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C0-R1
# RECOVER FROZEN FD002 PHASE-ADJUSTMENT PROVENANCE
#
# PURPOSE
# -------
# Identify the exact FIT-calibrated objects/functions used by
# the repaired FD002 phase-adjusted uncertainty chain.
#
# Diagnostic only.
# No model execution.
# No TEST labels.
# No threshold selection.
# No files written.
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R1 — PHASE-ADJUSTMENT PROVENANCE RECOVERY\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Confirm the final repaired objects
# ------------------------------------------------------------

required_r1 = [
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
]

print("\n--- FINAL REPAIRED OBJECTS ---")

for name in required_r1:
    print(
        f"{name:<50}:",
        name in globals()
    )


# ------------------------------------------------------------
# 2. Search compactly for relevant phase / FIT / uncertainty
#    objects without dumping the whole notebook namespace.
# ------------------------------------------------------------

keywords = [
    "phase",
    "uncert",
    "mc_std",
    "reference",
    "628fb",
    "628fbr",
]

candidate_rows = []


for name, obj in globals().items():

    lname = name.lower()

    if not any(
        key in lname
        for key in keywords
    ):
        continue

    # Keep FD002 / Step-6.28-related candidates preferentially.
    if (
        "fd002" not in lname
        and
        "628" not in lname
        and
        "phase" not in lname
    ):
        continue

    obj_type = type(obj).__name__

    shape = getattr(
        obj,
        "shape",
        None
    )

    length = None

    try:
        if not isinstance(
            obj,
            (
                str,
                bytes,
            )
        ):
            length = len(obj)
    except Exception:
        length = None


    candidate_rows.append(
        {
            "object_name": name,
            "type": obj_type,
            "shape": str(shape),
            "length": length,
        }
    )


candidates_r1 = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        "object_name"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- RELEVANT STORED OBJECT CANDIDATES ---"
)

display(
    candidates_r1
)


# ------------------------------------------------------------
# 3. Search specifically for FIT-related uncertainty objects
# ------------------------------------------------------------

fit_candidates = []


for name, obj in globals().items():

    lname = name.lower()

    if (
        "fit" in lname
        and
        (
            "uncert" in lname
            or
            "mc" in lname
            or
            "phase" in lname
            or
            "pred" in lname
        )
        and
        (
            "fd002" in lname
            or
            "628" in lname
        )
    ):

        fit_candidates.append(
            {
                "object_name": name,
                "type": type(obj).__name__,
                "shape": str(
                    getattr(
                        obj,
                        "shape",
                        None
                    )
                ),
            }
        )


fit_candidates_r1 = (
    pd.DataFrame(
        fit_candidates
    )
    .sort_values(
        "object_name"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- FIT-RELATED UNCERTAINTY / PHASE CANDIDATES ---"
)

display(
    fit_candidates_r1
)


# ------------------------------------------------------------
# 4. Search for callable helpers that may implement the
#    actual frozen phase-adjustment procedure.
# ------------------------------------------------------------

function_rows = []


for name, obj in globals().items():

    if not callable(obj):
        continue

    lname = name.lower()

    if not (
        "phase" in lname
        or
        (
            "uncert" in lname
            and
            "628" in lname
        )
        or
        "ecdf" in lname
        or
        "percentile" in lname
    ):
        continue

    try:
        signature = str(
            inspect.signature(
                obj
            )
        )
    except Exception:
        signature = "<unavailable>"


    function_rows.append(
        {
            "function_name": name,
            "signature": signature,
        }
    )


function_candidates_r1 = (
    pd.DataFrame(
        function_rows
    )
    .sort_values(
        "function_name"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- RELEVANT CALLABLE HELPERS ---"
)

display(
    function_candidates_r1
)


# ------------------------------------------------------------
# 5. Search dictionaries/contracts for useful provenance.
# ------------------------------------------------------------

contract_rows = []


for name, obj in globals().items():

    if not isinstance(
        obj,
        dict
    ):
        continue

    lname = name.lower()

    if not (
        "628" in lname
        or
        "fd002" in lname
    ):
        continue

    useful_keys = [
        key
        for key in obj.keys()
        if any(
            token in str(key).lower()
            for token in [
                "phase",
                "reference",
                "uncert",
                "fit",
                "recipe",
                "bin",
                "ecdf",
            ]
        )
    ]


    if useful_keys:

        contract_rows.append(
            {
                "object_name": name,
                "relevant_keys":
                    useful_keys,
            }
        )


print(
    "\n--- CONTRACT / DICTIONARY PROVENANCE ---"
)

for row in contract_rows:

    print(
        "\nObject:",
        row[
            "object_name"
        ]
    )

    print(
        "Relevant keys:",
        row[
            "relevant_keys"
        ]
    )


# ------------------------------------------------------------
# 6. Fingerprint the FINAL stored adjusted uncertainty.
# ------------------------------------------------------------

dev_adjusted_r1 = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


test_adjusted_r1 = np.asarray(
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


print(
    "\n--- FINAL STORED ADJUSTED-U FINGERPRINT ---"
)

for label, values in [
    (
        "DEV",
        dev_adjusted_r1
    ),
    (
        "TEST",
        test_adjusted_r1
    ),
]:

    print(
        f"{label:<5}",
        f"n={len(values)}",
        f"min={values.min():.12f}",
        f"q25={np.quantile(values, 0.25):.12f}",
        f"median={np.median(values):.12f}",
        f"mean={values.mean():.12f}",
        f"q75={np.quantile(values, 0.75):.12f}",
        f"max={values.max():.12f}",
    )


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Purpose                  : recover exact FIT calibration"
)

print(
    "Model executed           : False"
)

print(
    "MC dropout executed      : False"
)

print(
    "Official TEST labels read: False"
)

print(
    "Threshold selected       : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.28H-C0-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C0-R1
# PHASE-ADJUSTMENT PROVENANCE RECOVERY — CORRECTED VERSION
#
# PURPOSE
# -------
# Recover the exact FIT-calibrated provenance used by the
# frozen FD002 phase-adjusted uncertainty chain.
#
# IMPORTANT
# ---------
# Diagnostic only.
# No model training.
# No model inference.
# No MC-dropout rerun.
# No official TEST labels.
# No threshold selection.
# No files written.
# ============================================================

import numpy as np
import pandas as pd
import inspect

from IPython.display import display


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R1 — PHASE-ADJUSTMENT PROVENANCE RECOVERY\n"
    "============================================================"
)


# ============================================================
# 0. FREEZE A SAFE SNAPSHOT OF THE CURRENT NOTEBOOK NAMESPACE
#
# CRITICAL FIX:
# We DO NOT iterate directly over globals().items().
# That caused:
#
# RuntimeError: dictionary changed size during iteration
#
# Every namespace search below uses this immutable snapshot.
# ============================================================

GLOBAL_SNAPSHOT_628HC0R1 = tuple(
    globals().items()
)

print(
    "\n--- NAMESPACE SNAPSHOT ---"
)

print(
    "Objects captured:",
    len(
        GLOBAL_SNAPSHOT_628HC0R1
    )
)


# ============================================================
# 1. FINAL REPAIRED OBJECT CONTRACT
# ============================================================

required_r1 = [

    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",

    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",

    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
]


print(
    "\n--- FINAL REPAIRED OBJECTS ---"
)


missing_r1 = []


for name_r1 in required_r1:

    exists_r1 = (
        name_r1
        in
        globals()
    )

    print(
        f"{name_r1:<52}: "
        f"{exists_r1}"
    )

    if not exists_r1:

        missing_r1.append(
            name_r1
        )


if missing_r1:

    raise RuntimeError(
        "Required repaired FD002 objects are missing: "
        +
        str(
            missing_r1
        )
    )


# ============================================================
# 2. SEARCH FOR RELEVANT STORED OBJECTS
#
# We are looking for objects related to:
#
# - phase adjustment
# - uncertainty
# - MC std
# - calibration references
# - Step 6.28FB / 6.28FBR
# ============================================================

keywords_r1 = [

    "phase",

    "uncert",

    "mc_std",

    "reference",

    "628fb",

    "628fbr",
]


candidate_rows_r1 = []


for name_r1, obj_r1 in GLOBAL_SNAPSHOT_628HC0R1:

    lname_r1 = str(
        name_r1
    ).lower()


    if not any(
        keyword_r1
        in
        lname_r1

        for keyword_r1
        in
        keywords_r1
    ):

        continue


    # Prefer objects clearly associated with FD002 / Step 6.28.
    if not (
        "fd002"
        in
        lname_r1

        or

        "628"
        in
        lname_r1

        or

        "phase"
        in
        lname_r1
    ):

        continue


    obj_type_r1 = type(
        obj_r1
    ).__name__


    try:

        shape_r1 = getattr(
            obj_r1,
            "shape",
            None
        )

    except Exception:

        shape_r1 = "<unavailable>"


    try:

        if isinstance(
            obj_r1,
            (
                str,
                bytes,
            )
        ):

            length_r1 = None

        else:

            length_r1 = len(
                obj_r1
            )

    except Exception:

        length_r1 = None


    candidate_rows_r1.append(
        {

            "object_name":
                str(
                    name_r1
                ),

            "type":
                obj_type_r1,

            "shape":
                str(
                    shape_r1
                ),

            "length":
                length_r1,
        }
    )


if candidate_rows_r1:

    candidates_r1 = pd.DataFrame(
        candidate_rows_r1
    )

    candidates_r1 = (
        candidates_r1
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )

else:

    candidates_r1 = pd.DataFrame(
        columns=[
            "object_name",
            "type",
            "shape",
            "length",
        ]
    )


print(
    "\n--- RELEVANT STORED OBJECT CANDIDATES ---"
)

display(
    candidates_r1
)


# ============================================================
# 3. SEARCH SPECIFICALLY FOR FIT-RELATED UNCERTAINTY /
#    PHASE-CALIBRATION OBJECTS
# ============================================================

fit_candidate_rows_r1 = []


for name_r1, obj_r1 in GLOBAL_SNAPSHOT_628HC0R1:

    lname_r1 = str(
        name_r1
    ).lower()


    relevant_fit_r1 = (
        "fit"
        in
        lname_r1
    )


    relevant_uncertainty_r1 = (

        "uncert"
        in
        lname_r1

        or

        "mc"
        in
        lname_r1

        or

        "phase"
        in
        lname_r1

        or

        "pred"
        in
        lname_r1

        or

        "reference"
        in
        lname_r1
    )


    relevant_project_r1 = (

        "fd002"
        in
        lname_r1

        or

        "628"
        in
        lname_r1
    )


    if not (
        relevant_fit_r1
        and
        relevant_uncertainty_r1
        and
        relevant_project_r1
    ):

        continue


    try:

        shape_r1 = getattr(
            obj_r1,
            "shape",
            None
        )

    except Exception:

        shape_r1 = "<unavailable>"


    fit_candidate_rows_r1.append(
        {

            "object_name":
                str(
                    name_r1
                ),

            "type":
                type(
                    obj_r1
                ).__name__,

            "shape":
                str(
                    shape_r1
                ),
        }
    )


if fit_candidate_rows_r1:

    fit_candidates_r1 = pd.DataFrame(
        fit_candidate_rows_r1
    )

    fit_candidates_r1 = (
        fit_candidates_r1
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )

else:

    fit_candidates_r1 = pd.DataFrame(
        columns=[
            "object_name",
            "type",
            "shape",
        ]
    )


print(
    "\n--- FIT-RELATED UNCERTAINTY / PHASE CANDIDATES ---"
)

display(
    fit_candidates_r1
)


# ============================================================
# 4. SEARCH FOR CALLABLE HELPERS
#
# These may contain the exact original implementation of:
#
# - phase adjustment
# - uncertainty calibration
# - ECDF transformation
# - percentile / phase binning
# ============================================================

function_candidate_rows_r1 = []


for name_r1, obj_r1 in GLOBAL_SNAPSHOT_628HC0R1:

    try:

        callable_r1 = callable(
            obj_r1
        )

    except Exception:

        callable_r1 = False


    if not callable_r1:

        continue


    lname_r1 = str(
        name_r1
    ).lower()


    relevant_function_r1 = (

        "phase"
        in
        lname_r1

        or

        "ecdf"
        in
        lname_r1

        or

        "percentile"
        in
        lname_r1

        or

        "calibr"
        in
        lname_r1

        or

        (
            "uncert"
            in
            lname_r1

            and

            (
                "628"
                in
                lname_r1

                or

                "fd002"
                in
                lname_r1
            )
        )
    )


    if not relevant_function_r1:

        continue


    try:

        signature_r1 = str(
            inspect.signature(
                obj_r1
            )
        )

    except Exception:

        signature_r1 = "<signature unavailable>"


    function_candidate_rows_r1.append(
        {

            "function_name":
                str(
                    name_r1
                ),

            "type":
                type(
                    obj_r1
                ).__name__,

            "signature":
                signature_r1,
        }
    )


if function_candidate_rows_r1:

    function_candidates_r1 = pd.DataFrame(
        function_candidate_rows_r1
    )

    function_candidates_r1 = (
        function_candidates_r1
        .sort_values(
            "function_name"
        )
        .reset_index(
            drop=True
        )
    )

else:

    function_candidates_r1 = pd.DataFrame(
        columns=[
            "function_name",
            "type",
            "signature",
        ]
    )


print(
    "\n--- RELEVANT CALLABLE HELPERS ---"
)

display(
    function_candidates_r1
)


# ============================================================
# 5. SEARCH DICTIONARIES / CONTRACTS FOR CALIBRATION PROVENANCE
# ============================================================

contract_rows_r1 = []


provenance_tokens_r1 = [

    "phase",

    "reference",

    "uncert",

    "fit",

    "recipe",

    "bin",

    "ecdf",

    "calibr",

    "percent",

    "std",
]


for name_r1, obj_r1 in GLOBAL_SNAPSHOT_628HC0R1:

    if not isinstance(
        obj_r1,
        dict
    ):

        continue


    lname_r1 = str(
        name_r1
    ).lower()


    if not (
        "628"
        in
        lname_r1

        or

        "fd002"
        in
        lname_r1
    ):

        continue


    try:

        object_keys_r1 = list(
            obj_r1.keys()
        )

    except Exception:

        continue


    useful_keys_r1 = []


    for key_r1 in object_keys_r1:

        key_text_r1 = str(
            key_r1
        ).lower()


        if any(
            token_r1
            in
            key_text_r1

            for token_r1
            in
            provenance_tokens_r1
        ):

            useful_keys_r1.append(
                key_r1
            )


    if useful_keys_r1:

        contract_rows_r1.append(
            {

                "object_name":
                    str(
                        name_r1
                    ),

                "object":
                    obj_r1,

                "relevant_keys":
                    useful_keys_r1,
            }
        )


print(
    "\n--- CONTRACT / DICTIONARY PROVENANCE ---"
)


if not contract_rows_r1:

    print(
        "No matching dictionary provenance objects found."
    )


for contract_r1 in contract_rows_r1:

    object_name_r1 = contract_r1[
        "object_name"
    ]

    object_r1 = contract_r1[
        "object"
    ]

    relevant_keys_r1 = contract_r1[
        "relevant_keys"
    ]


    print(
        "\nObject:",
        object_name_r1
    )


    print(
        "Relevant keys:",
        relevant_keys_r1
    )


    for key_r1 in relevant_keys_r1:

        try:

            value_r1 = object_r1[
                key_r1
            ]

        except Exception as exc_r1:

            print(
                f"  {key_r1}: "
                f"<could not read: {exc_r1}>"
            )

            continue


        # ----------------------------------------------------
        # Compact display — avoid printing giant arrays.
        # ----------------------------------------------------

        if isinstance(
            value_r1,
            np.ndarray
        ):

            finite_r1 = bool(
                np.isfinite(
                    value_r1
                ).all()
            ) if np.issubdtype(
                value_r1.dtype,
                np.number
            ) else None


            print(
                f"  {key_r1}: "
                f"ndarray "
                f"shape={value_r1.shape} "
                f"dtype={value_r1.dtype} "
                f"finite={finite_r1}"
            )


        elif isinstance(
            value_r1,
            pd.DataFrame
        ):

            print(
                f"  {key_r1}: "
                f"DataFrame "
                f"shape={value_r1.shape} "
                f"columns={list(value_r1.columns)}"
            )


        elif isinstance(
            value_r1,
            pd.Series
        ):

            print(
                f"  {key_r1}: "
                f"Series "
                f"shape={value_r1.shape} "
                f"name={value_r1.name}"
            )


        elif callable(
            value_r1
        ):

            try:

                value_signature_r1 = str(
                    inspect.signature(
                        value_r1
                    )
                )

            except Exception:

                value_signature_r1 = (
                    "<signature unavailable>"
                )


            print(
                f"  {key_r1}: "
                f"callable "
                f"{getattr(value_r1, '__name__', type(value_r1).__name__)}"
                f"{value_signature_r1}"
            )


        else:

            value_repr_r1 = repr(
                value_r1
            )


            if len(
                value_repr_r1
            ) > 500:

                value_repr_r1 = (
                    value_repr_r1[
                        :500
                    ]
                    +
                    " ... <truncated>"
                )


            print(
                f"  {key_r1}: "
                f"{value_repr_r1}"
            )


# ============================================================
# 6. SEARCH FOR OBJECTS WITH LENGTHS THAT MAY MATCH
#    IMPORTANT CALIBRATION POPULATIONS
#
# Known relevant FD002 counts include:
#
# DEV windows       = 8367
# FIT windows       = 37852
# DEV engines       = 52
# TEST engines      = 259
# phase bins        = 5
#
# This may reveal hidden calibration references.
# ============================================================

interesting_lengths_r1 = {
    5,
    52,
    259,
    8367,
    37852,
}


length_match_rows_r1 = []


for name_r1, obj_r1 in GLOBAL_SNAPSHOT_628HC0R1:

    lname_r1 = str(
        name_r1
    ).lower()


    # Restrict search so output stays useful.
    if not (
        "628"
        in
        lname_r1

        or

        "fd002"
        in
        lname_r1

        or

        "phase"
        in
        lname_r1

        or

        "uncert"
        in
        lname_r1

        or

        "reference"
        in
        lname_r1
    ):

        continue


    try:

        object_length_r1 = len(
            obj_r1
        )

    except Exception:

        continue


    if object_length_r1 not in interesting_lengths_r1:

        continue


    try:

        shape_r1 = getattr(
            obj_r1,
            "shape",
            None
        )

    except Exception:

        shape_r1 = "<unavailable>"


    length_match_rows_r1.append(
        {

            "object_name":
                str(
                    name_r1
                ),

            "type":
                type(
                    obj_r1
                ).__name__,

            "length":
                object_length_r1,

            "shape":
                str(
                    shape_r1
                ),
        }
    )


if length_match_rows_r1:

    length_matches_r1 = pd.DataFrame(
        length_match_rows_r1
    )

    length_matches_r1 = (
        length_matches_r1
        .sort_values(
            [
                "length",
                "object_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    length_matches_r1 = pd.DataFrame(
        columns=[
            "object_name",
            "type",
            "length",
            "shape",
        ]
    )


print(
    "\n--- CALIBRATION-POPULATION LENGTH MATCHES ---"
)

display(
    length_matches_r1
)


# ============================================================
# 7. FINGERPRINT FINAL STORED ADJUSTED UNCERTAINTY
# ============================================================

dev_adjusted_r1 = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


test_adjusted_r1 = np.asarray(
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


if len(
    dev_adjusted_r1
) != 8367:

    raise RuntimeError(
        "Unexpected DEV adjusted-uncertainty length: "
        +
        str(
            len(
                dev_adjusted_r1
            )
        )
    )


if len(
    test_adjusted_r1
) != 259:

    raise RuntimeError(
        "Unexpected TEST adjusted-uncertainty length: "
        +
        str(
            len(
                test_adjusted_r1
            )
        )
    )


if not np.isfinite(
    dev_adjusted_r1
).all():

    raise RuntimeError(
        "Stored DEV adjusted uncertainty contains "
        "non-finite values."
    )


if not np.isfinite(
    test_adjusted_r1
).all():

    raise RuntimeError(
        "Stored TEST adjusted uncertainty contains "
        "non-finite values."
    )


print(
    "\n--- FINAL STORED ADJUSTED-U FINGERPRINT ---"
)


for label_r1, values_r1 in [

    (
        "DEV",
        dev_adjusted_r1
    ),

    (
        "TEST",
        test_adjusted_r1
    ),
]:

    print(
        f"{label_r1:<5}"
        f" n={len(values_r1)}"
        f" min={np.min(values_r1):.12f}"
        f" q25={np.quantile(values_r1, 0.25):.12f}"
        f" median={np.median(values_r1):.12f}"
        f" mean={np.mean(values_r1):.12f}"
        f" q75={np.quantile(values_r1, 0.75):.12f}"
        f" q95={np.quantile(values_r1, 0.95):.12f}"
        f" max={np.max(values_r1):.12f}"
    )


# ============================================================
# 8. G1 FINGERPRINT
#
# Useful for checking whether the adjusted-U object is indeed
# the exact parent of the frozen G1 object.
# ============================================================

dev_g1_r1 = np.asarray(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)


print(
    "\n--- STORED G1 FINGERPRINT ---"
)


print(
    "DEV G1 count  :",
    len(
        dev_g1_r1
    )
)


print(
    "DEV G1 finite :",
    bool(
        np.isfinite(
            dev_g1_r1
        ).all()
    )
)


print(
    "DEV G1 min    :",
    float(
        np.min(
            dev_g1_r1
        )
    )
)


print(
    "DEV G1 median :",
    float(
        np.median(
            dev_g1_r1
        )
    )
)


print(
    "DEV G1 mean   :",
    float(
        np.mean(
            dev_g1_r1
        )
    )
)


print(
    "DEV G1 max    :",
    float(
        np.max(
            dev_g1_r1
        )
    )
)


# ============================================================
# 9. FINAL DIAGNOSTIC CONTRACT
# ============================================================

R1_PROVENANCE_DIAGNOSTIC_628HC0R1 = {

    "namespace_snapshot_size":
        len(
            GLOBAL_SNAPSHOT_628HC0R1
        ),

    "stored_candidate_count":
        len(
            candidates_r1
        ),

    "fit_candidate_count":
        len(
            fit_candidates_r1
        ),

    "callable_candidate_count":
        len(
            function_candidates_r1
        ),

    "dictionary_candidate_count":
        len(
            contract_rows_r1
        ),

    "length_match_count":
        len(
            length_matches_r1
        ),

    "dev_adjusted_count":
        len(
            dev_adjusted_r1
        ),

    "test_adjusted_count":
        len(
            test_adjusted_r1
        ),

    "dev_adjusted_finite":
        bool(
            np.isfinite(
                dev_adjusted_r1
            ).all()
        ),

    "test_adjusted_finite":
        bool(
            np.isfinite(
                test_adjusted_r1
            ).all()
        ),

    "model_executed":
        False,

    "mc_dropout_executed":
        False,

    "official_test_labels_read":
        False,

    "threshold_selected":
        False,

    "files_written":
        False,
}


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R1 FINAL SUMMARY\n"
    "============================================================"
)


print(
    "Purpose                     : "
    "recover exact FIT-calibrated phase provenance"
)


print(
    "Stored candidate objects    :",
    len(
        candidates_r1
    )
)


print(
    "FIT-related candidates      :",
    len(
        fit_candidates_r1
    )
)


print(
    "Relevant callable helpers   :",
    len(
        function_candidates_r1
    )
)


print(
    "Relevant dictionary objects :",
    len(
        contract_rows_r1
    )
)


print(
    "Calibration-length matches  :",
    len(
        length_matches_r1
    )
)


print(
    "DEV adjusted-U finite       :",
    bool(
        np.isfinite(
            dev_adjusted_r1
        ).all()
    )
)


print(
    "TEST adjusted-U finite      :",
    bool(
        np.isfinite(
            test_adjusted_r1
        ).all()
    )
)


print(
    "Model executed              : False"
)


print(
    "MC dropout executed         : False"
)


print(
    "Official TEST labels read   : False"
)


print(
    "Threshold selected          : False"
)


print(
    "Files written               : False"
)


print(
    "\n=== END STEP 6.28H-C0-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C0-R2
# EXACT FD002 CLEAN RELIABILITY-CHAIN CLOSURE
#
# PURPOSE
# -------
# Reproduce the already-frozen FD002 clean reliability chain
# using the exact recovered FIT-calibrated phase-adjustment
# implementation.
#
# This cell:
#   1. reproduces DEV and TEST phase-adjusted uncertainty,
#   2. reproduces G1,
#   3. reconstructs the corrected weighted-midrank G2,
#   4. checks the complete clean reliability chain,
#   5. freezes reusable reliability functions for later
#      corrupted-scenario evaluation.
#
# IMPORTANT
# ---------
# - No model retraining.
# - No new methodology selection.
# - No threshold selection.
# - No official TEST labels.
# - No TEST scoring.
# - No corruption applied.
# - No files written.
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R2 — EXACT CLEAN RELIABILITY-CHAIN CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECT CONTRACT
# ============================================================

required_objects_628hc0r2 = [

    # Exact recovered phase-adjustment implementation
    "apply_phase_adjustment_628fbr1",

    # Deterministic clean predictions
    "DEV_PREDICTIONS_FD002_628EB",
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",

    # Clean MC-dropout STD
    "DEV_MC_STD_FD002_628FAR2",
    "TEST_MC_STD_FD002_628FAR2",

    # Frozen adjusted uncertainty
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",

    # Frozen G1 implementation and values
    "apply_g1_right_ecdf_628fc",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",

    # Corrected G2 reference
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",

    # Frozen final selective-risk values
    "FD002_DEV_SELECTIVE_RISK_628FCR2",
    "FD002_TEST_SELECTIVE_RISK_628FCR2",

    # Frozen methodology evidence
    "FD002_UNCERTAINTY_RECIPE_628FBR1",
    "FD002_CLEAN_RELIABILITY_CONTRACT_628FCR3",
]


print(
    "\n--- REQUIRED OBJECTS ---"
)


missing_objects_628hc0r2 = []


for name_628hc0r2 in required_objects_628hc0r2:

    exists_628hc0r2 = (
        name_628hc0r2
        in
        globals()
    )

    print(
        f"{name_628hc0r2:<55}: "
        f"{exists_628hc0r2}"
    )

    if not exists_628hc0r2:

        missing_objects_628hc0r2.append(
            name_628hc0r2
        )


if missing_objects_628hc0r2:

    raise RuntimeError(
        "Required object(s) missing: "
        +
        str(
            missing_objects_628hc0r2
        )
    )


# ============================================================
# 2. FREEZE INPUT ARRAYS
# ============================================================

dev_prediction_628hc0r2 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64
).reshape(-1)


test_prediction_628hc0r2 = np.asarray(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)


dev_mc_std_628hc0r2 = np.asarray(
    DEV_MC_STD_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)


test_mc_std_628hc0r2 = np.asarray(
    TEST_MC_STD_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)


stored_dev_adjusted_628hc0r2 = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


stored_test_adjusted_628hc0r2 = np.asarray(
    FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


stored_dev_g1_628hc0r2 = np.asarray(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)


stored_test_g1_628hc0r2 = np.asarray(
    FD002_TEST_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)


stored_dev_final_risk_628hc0r2 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


stored_test_final_risk_628hc0r2 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


print(
    "\n--- POPULATION CONTRACT ---"
)

print(
    "DEV predictions :",
    len(
        dev_prediction_628hc0r2
    )
)

print(
    "DEV MC STD      :",
    len(
        dev_mc_std_628hc0r2
    )
)

print(
    "TEST predictions:",
    len(
        test_prediction_628hc0r2
    )
)

print(
    "TEST MC STD     :",
    len(
        test_mc_std_628hc0r2
    )
)


if len(
    dev_prediction_628hc0r2
) != 8367:

    raise RuntimeError(
        "Unexpected DEV prediction count."
    )


if len(
    dev_mc_std_628hc0r2
) != 8367:

    raise RuntimeError(
        "Unexpected DEV MC-STD count."
    )


if len(
    test_prediction_628hc0r2
) != 259:

    raise RuntimeError(
        "Unexpected TEST prediction count."
    )


if len(
    test_mc_std_628hc0r2
) != 259:

    raise RuntimeError(
        "Unexpected TEST MC-STD count."
    )


# ============================================================
# 3. EXACT PHASE-ADJUSTMENT REPRODUCTION
#
# We now use the ACTUAL original frozen implementation:
#
# apply_phase_adjustment_628fbr1(prediction, mc_std)
#
# Its calibration reference was FIT-only.
# ============================================================

reproduced_dev_adjusted_628hc0r2 = np.asarray(
    apply_phase_adjustment_628fbr1(
        dev_prediction_628hc0r2,
        dev_mc_std_628hc0r2
    ),
    dtype=np.float64
).reshape(-1)


reproduced_test_adjusted_628hc0r2 = np.asarray(
    apply_phase_adjustment_628fbr1(
        test_prediction_628hc0r2,
        test_mc_std_628hc0r2
    ),
    dtype=np.float64
).reshape(-1)


dev_phase_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_dev_adjusted_628hc0r2
            -
            stored_dev_adjusted_628hc0r2
        )
    )
)


test_phase_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_test_adjusted_628hc0r2
            -
            stored_test_adjusted_628hc0r2
        )
    )
)


dev_phase_exact_628hc0r2 = bool(
    dev_phase_max_error_628hc0r2
    <=
    1e-12
)


test_phase_exact_628hc0r2 = bool(
    test_phase_max_error_628hc0r2
    <=
    1e-12
)


print(
    "\n--- EXACT PHASE-ADJUSTMENT REPRODUCTION ---"
)

print(
    "DEV maximum absolute error :",
    dev_phase_max_error_628hc0r2
)

print(
    "TEST maximum absolute error:",
    test_phase_max_error_628hc0r2
)

print(
    "DEV exact <= 1e-12         :",
    dev_phase_exact_628hc0r2
)

print(
    "TEST exact <= 1e-12        :",
    test_phase_exact_628hc0r2
)


if not dev_phase_exact_628hc0r2:

    raise RuntimeError(
        "Exact DEV phase-adjusted uncertainty "
        "was not reproduced."
    )


if not test_phase_exact_628hc0r2:

    raise RuntimeError(
        "Exact TEST phase-adjusted uncertainty "
        "was not reproduced."
    )


# ============================================================
# 4. G1 REPRODUCTION
#
# Frozen G1:
# right empirical CDF against all clean DEV windows.
# ============================================================

reproduced_dev_g1_628hc0r2 = np.asarray(
    apply_g1_right_ecdf_628fc(
        reproduced_dev_adjusted_628hc0r2
    ),
    dtype=np.float64
).reshape(-1)


reproduced_test_g1_628hc0r2 = np.asarray(
    apply_g1_right_ecdf_628fc(
        reproduced_test_adjusted_628hc0r2
    ),
    dtype=np.float64
).reshape(-1)


dev_g1_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_dev_g1_628hc0r2
            -
            stored_dev_g1_628hc0r2
        )
    )
)


test_g1_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_test_g1_628hc0r2
            -
            stored_test_g1_628hc0r2
        )
    )
)


dev_g1_exact_628hc0r2 = bool(
    dev_g1_max_error_628hc0r2
    <=
    1e-12
)


test_g1_exact_628hc0r2 = bool(
    test_g1_max_error_628hc0r2
    <=
    1e-12
)


print(
    "\n--- G1 REPRODUCTION ---"
)

print(
    "DEV G1 maximum absolute error :",
    dev_g1_max_error_628hc0r2
)

print(
    "TEST G1 maximum absolute error:",
    test_g1_max_error_628hc0r2
)

print(
    "DEV G1 exact <= 1e-12         :",
    dev_g1_exact_628hc0r2
)

print(
    "TEST G1 exact <= 1e-12        :",
    test_g1_exact_628hc0r2
)


if not dev_g1_exact_628hc0r2:

    raise RuntimeError(
        "Frozen DEV G1 values were not reproduced."
    )


if not test_g1_exact_628hc0r2:

    raise RuntimeError(
        "Frozen TEST G1 values were not reproduced."
    )


# ============================================================
# 5. CORRECTED G2 REFERENCE CONTRACT
#
# Correct methodology:
#
# - all clean DEV candidate endpoints
# - engine-balanced weights
# - weighted empirical MID-RANK CDF
# ============================================================

g2_reference_values_628hc0r2 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


g2_reference_weights_628hc0r2 = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


if len(
    g2_reference_values_628hc0r2
) != 8367:

    raise RuntimeError(
        "Unexpected G2 reference-value count."
    )


if len(
    g2_reference_weights_628hc0r2
) != 8367:

    raise RuntimeError(
        "Unexpected G2 reference-weight count."
    )


if not np.isfinite(
    g2_reference_values_628hc0r2
).all():

    raise RuntimeError(
        "G2 reference values contain "
        "non-finite entries."
    )


if not np.isfinite(
    g2_reference_weights_628hc0r2
).all():

    raise RuntimeError(
        "G2 reference weights contain "
        "non-finite entries."
    )


if np.any(
    g2_reference_weights_628hc0r2
    <
    0
):

    raise RuntimeError(
        "G2 reference contains negative weights."
    )


g2_weight_sum_628hc0r2 = float(
    np.sum(
        g2_reference_weights_628hc0r2
    )
)


print(
    "\n--- CORRECTED G2 REFERENCE CONTRACT ---"
)

print(
    "Reference values :",
    len(
        g2_reference_values_628hc0r2
    )
)

print(
    "Reference weights:",
    len(
        g2_reference_weights_628hc0r2
    )
)

print(
    "Weight sum       :",
    g2_weight_sum_628hc0r2
)


if not np.isclose(
    g2_weight_sum_628hc0r2,
    1.0,
    atol=1e-12,
    rtol=0.0
):

    raise RuntimeError(
        "G2 reference weights do not sum to 1."
    )


# ============================================================
# 6. EXACT WEIGHTED EMPIRICAL MID-RANK CDF
#
# For query x:
#
# F_mid(x)
# =
# weight(reference < x)
# +
# 0.5 * weight(reference == x)
#
# Reference values and their weights are sorted together.
# ============================================================

def apply_fd002_g2_weighted_midrank_628hc0r2(
    values
):

    values_local = np.asarray(
        values,
        dtype=np.float64
    ).reshape(-1)


    if not np.isfinite(
        values_local
    ).all():

        raise ValueError(
            "G2 input contains non-finite values."
        )


    reference_local = np.asarray(
        G2_REFERENCE_VALUES_FD002_628FCR2,
        dtype=np.float64
    ).reshape(-1)


    weights_local = np.asarray(
        G2_REFERENCE_WEIGHTS_FD002_628FCR2,
        dtype=np.float64
    ).reshape(-1)


    order_local = np.argsort(
        reference_local,
        kind="mergesort"
    )


    reference_sorted_local = (
        reference_local[
            order_local
        ]
    )


    weights_sorted_local = (
        weights_local[
            order_local
        ]
    )


    prefix_local = np.concatenate(
        [
            np.array(
                [0.0],
                dtype=np.float64
            ),
            np.cumsum(
                weights_sorted_local,
                dtype=np.float64
            )
        ]
    )


    left_idx_local = np.searchsorted(
        reference_sorted_local,
        values_local,
        side="left"
    )


    right_idx_local = np.searchsorted(
        reference_sorted_local,
        values_local,
        side="right"
    )


    weight_less_local = (
        prefix_local[
            left_idx_local
        ]
    )


    weight_equal_local = (
        prefix_local[
            right_idx_local
        ]
        -
        prefix_local[
            left_idx_local
        ]
    )


    result_local = (
        weight_less_local
        +
        0.5
        *
        weight_equal_local
    )


    return np.clip(
        result_local,
        0.0,
        1.0
    )


# ============================================================
# 7. G2 / FINAL SELECTIVE-RISK REPRODUCTION
# ============================================================

reproduced_dev_final_risk_628hc0r2 = (
    apply_fd002_g2_weighted_midrank_628hc0r2(
        reproduced_dev_g1_628hc0r2
    )
)


reproduced_test_final_risk_628hc0r2 = (
    apply_fd002_g2_weighted_midrank_628hc0r2(
        reproduced_test_g1_628hc0r2
    )
)


dev_final_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_dev_final_risk_628hc0r2
            -
            stored_dev_final_risk_628hc0r2
        )
    )
)


test_final_max_error_628hc0r2 = float(
    np.max(
        np.abs(
            reproduced_test_final_risk_628hc0r2
            -
            stored_test_final_risk_628hc0r2
        )
    )
)


dev_final_exact_628hc0r2 = bool(
    dev_final_max_error_628hc0r2
    <=
    1e-12
)


test_final_exact_628hc0r2 = bool(
    test_final_max_error_628hc0r2
    <=
    1e-12
)


print(
    "\n--- FINAL SELECTIVE-RISK REPRODUCTION ---"
)

print(
    "DEV final-risk maximum absolute error :",
    dev_final_max_error_628hc0r2
)

print(
    "TEST final-risk maximum absolute error:",
    test_final_max_error_628hc0r2
)

print(
    "DEV final risk exact <= 1e-12         :",
    dev_final_exact_628hc0r2
)

print(
    "TEST final risk exact <= 1e-12        :",
    test_final_exact_628hc0r2
)


if not dev_final_exact_628hc0r2:

    raise RuntimeError(
        "Corrected DEV final selective risk "
        "was not exactly reproduced."
    )


if not test_final_exact_628hc0r2:

    raise RuntimeError(
        "Corrected TEST final selective risk "
        "was not exactly reproduced."
    )


# ============================================================
# 8. FREEZE REUSABLE EXACT RELIABILITY FUNCTIONS
#
# These functions will be used later for corrupted predictions
# and corrupted MC-dropout uncertainty.
# ============================================================

def fd002_phase_adjusted_uncertainty_628hc0r2(
    deterministic_prediction,
    mc_std
):

    prediction_local = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    ).reshape(-1)


    mc_std_local = np.asarray(
        mc_std,
        dtype=np.float64
    ).reshape(-1)


    if len(
        prediction_local
    ) != len(
        mc_std_local
    ):

        raise ValueError(
            "Prediction and MC-STD lengths differ."
        )


    adjusted_local = np.asarray(
        apply_phase_adjustment_628fbr1(
            prediction_local,
            mc_std_local
        ),
        dtype=np.float64
    ).reshape(-1)


    return adjusted_local


def fd002_g1_uncertainty_risk_628hc0r2(
    adjusted_uncertainty
):

    adjusted_local = np.asarray(
        adjusted_uncertainty,
        dtype=np.float64
    ).reshape(-1)


    return np.asarray(
        apply_g1_right_ecdf_628fc(
            adjusted_local
        ),
        dtype=np.float64
    ).reshape(-1)


def fd002_final_selective_risk_628hc0r2(
    g1_risk
):

    return (
        apply_fd002_g2_weighted_midrank_628hc0r2(
            g1_risk
        )
    )


def fd002_complete_reliability_chain_628hc0r2(
    deterministic_prediction,
    mc_std
):

    adjusted_local = (
        fd002_phase_adjusted_uncertainty_628hc0r2(
            deterministic_prediction,
            mc_std
        )
    )


    g1_local = (
        fd002_g1_uncertainty_risk_628hc0r2(
            adjusted_local
        )
    )


    final_risk_local = (
        fd002_final_selective_risk_628hc0r2(
            g1_local
        )
    )


    return {
        "adjusted_uncertainty":
            adjusted_local,

        "g1_risk":
            g1_local,

        "final_selective_risk":
            final_risk_local,
    }


# ============================================================
# 9. END-TO-END CLEAN CHAIN RECHECK
# ============================================================

dev_chain_recheck_628hc0r2 = (
    fd002_complete_reliability_chain_628hc0r2(
        dev_prediction_628hc0r2,
        dev_mc_std_628hc0r2
    )
)


test_chain_recheck_628hc0r2 = (
    fd002_complete_reliability_chain_628hc0r2(
        test_prediction_628hc0r2,
        test_mc_std_628hc0r2
    )
)


end_to_end_checks_628hc0r2 = pd.DataFrame(
    [
        {
            "population":
                "DEV",

            "adjusted_u_max_error":
                float(
                    np.max(
                        np.abs(
                            dev_chain_recheck_628hc0r2[
                                "adjusted_uncertainty"
                            ]
                            -
                            stored_dev_adjusted_628hc0r2
                        )
                    )
                ),

            "g1_max_error":
                float(
                    np.max(
                        np.abs(
                            dev_chain_recheck_628hc0r2[
                                "g1_risk"
                            ]
                            -
                            stored_dev_g1_628hc0r2
                        )
                    )
                ),

            "final_risk_max_error":
                float(
                    np.max(
                        np.abs(
                            dev_chain_recheck_628hc0r2[
                                "final_selective_risk"
                            ]
                            -
                            stored_dev_final_risk_628hc0r2
                        )
                    )
                ),
        },

        {
            "population":
                "TEST",

            "adjusted_u_max_error":
                float(
                    np.max(
                        np.abs(
                            test_chain_recheck_628hc0r2[
                                "adjusted_uncertainty"
                            ]
                            -
                            stored_test_adjusted_628hc0r2
                        )
                    )
                ),

            "g1_max_error":
                float(
                    np.max(
                        np.abs(
                            test_chain_recheck_628hc0r2[
                                "g1_risk"
                            ]
                            -
                            stored_test_g1_628hc0r2
                        )
                    )
                ),

            "final_risk_max_error":
                float(
                    np.max(
                        np.abs(
                            test_chain_recheck_628hc0r2[
                                "final_selective_risk"
                            ]
                            -
                            stored_test_final_risk_628hc0r2
                        )
                    )
                ),
        },
    ]
)


end_to_end_checks_628hc0r2[
    "all_exact_1e12"
] = (

    (
        end_to_end_checks_628hc0r2[
            "adjusted_u_max_error"
        ]
        <=
        1e-12
    )

    &

    (
        end_to_end_checks_628hc0r2[
            "g1_max_error"
        ]
        <=
        1e-12
    )

    &

    (
        end_to_end_checks_628hc0r2[
            "final_risk_max_error"
        ]
        <=
        1e-12
    )
)


print(
    "\n--- END-TO-END CLEAN RELIABILITY CHAIN ---"
)

display(
    end_to_end_checks_628hc0r2
)


if not bool(
    end_to_end_checks_628hc0r2[
        "all_exact_1e12"
    ].all()
):

    raise RuntimeError(
        "The complete frozen clean reliability chain "
        "was not reproduced exactly."
    )


# ============================================================
# 10. FREEZE CLOSURE CONTRACT
# ============================================================

FD002_CORRUPTED_RELIABILITY_BRIDGE_CONTRACT_628HC0R2 = {

    "phase_recipe":
        "det_pred_pct_b5",

    "phase_reference":
        "FD002_FIT_only",

    "phase_bins":
        5,

    "phase_assignment":
        "np.digitize_right=True",

    "base_uncertainty":
        "MC_dropout_STD",

    "mc_std_ddof":
        0,

    "within_phase_mapping":
        "empirical_midrank_percentile",

    "g1_reference":
        "all_clean_FD002_DEV_windows",

    "g1_transform":
        "right_empirical_CDF",

    "g2_reference":
        (
            "all_clean_DEV_windows_as_"
            "possible_truncated_endpoints"
        ),

    "g2_weighting":
        "equal_total_engine_weighting",

    "g2_transform":
        "weighted_empirical_midrank_CDF",

    "dev_population":
        8367,

    "test_population":
        259,

    "dev_phase_exact":
        dev_phase_exact_628hc0r2,

    "test_phase_exact":
        test_phase_exact_628hc0r2,

    "dev_g1_exact":
        dev_g1_exact_628hc0r2,

    "test_g1_exact":
        test_g1_exact_628hc0r2,

    "dev_final_risk_exact":
        dev_final_exact_628hc0r2,

    "test_final_risk_exact":
        test_final_exact_628hc0r2,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "threshold_selected":
        False,

    "corruption_applied":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C0-R2 FINAL SUMMARY\n"
    "============================================================"
)


print(
    "Recovered phase reference    : FD002 FIT only"
)

print(
    "Phase bins                   : 5"
)

print(
    "Phase assignment             : right=True"
)

print(
    "Within-phase transform       : empirical mid-rank percentile"
)

print(
    "MC STD convention            : ddof=0"
)

print(
    "DEV phase adjustment exact   :",
    dev_phase_exact_628hc0r2
)

print(
    "TEST phase adjustment exact  :",
    test_phase_exact_628hc0r2
)

print(
    "DEV G1 exact                 :",
    dev_g1_exact_628hc0r2
)

print(
    "TEST G1 exact                :",
    test_g1_exact_628hc0r2
)

print(
    "DEV final risk exact         :",
    dev_final_exact_628hc0r2
)

print(
    "TEST final risk exact        :",
    test_final_exact_628hc0r2
)

print(
    "Complete DEV chain exact     :",
    bool(
        end_to_end_checks_628hc0r2
        .loc[
            end_to_end_checks_628hc0r2[
                "population"
            ]
            ==
            "DEV",
            "all_exact_1e12"
        ]
        .iloc[
            0
        ]
    )
)

print(
    "Complete TEST chain exact    :",
    bool(
        end_to_end_checks_628hc0r2
        .loc[
            end_to_end_checks_628hc0r2[
                "population"
            ]
            ==
            "TEST",
            "all_exact_1e12"
        ]
        .iloc[
            0
        ]
    )
)

print(
    "Official TEST labels read    : False"
)

print(
    "TEST scored                  : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Corruption applied           : False"
)

print(
    "Model retrained              : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.28H-C0-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C1-A
# CORRUPTED MC-DROPOUT BRIDGE PROVENANCE PRE-FLIGHT
#
# PURPOSE
# -------
# Identify the exact already-existing helpers and objects used
# to construct corrupted FD002 DEV inputs and perform inference.
#
# No corruption experiment is executed here.
# No model is retrained.
# No TEST labels are read.
# No files are written.
# ============================================================

import inspect
import numpy as np
import pandas as pd
from IPython.display import display


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-A — CORRUPTED MC-DROPOUT BRIDGE PRE-FLIGHT\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED FROZEN OBJECTS
# ============================================================

required_628hc1a = [
    "MODEL_FD002_628EB",
    "X_DEV_FD002_628D",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "FD002_DEV_CORRUPTION_RESPONSE_628HB",
    "FD002_CORRUPTED_RELIABILITY_BRIDGE_CONTRACT_628HC0R2",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

missing_628hc1a = []

for name_628hc1a in required_628hc1a:

    exists_628hc1a = name_628hc1a in globals()

    print(
        f"{name_628hc1a:<55}: "
        f"{exists_628hc1a}"
    )

    if not exists_628hc1a:
        missing_628hc1a.append(name_628hc1a)


if missing_628hc1a:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_628hc1a)
    )


# ============================================================
# 2. SNAPSHOT THE NAMESPACE
#
# Important:
# We copy globals() before iterating so that we do NOT repeat
# the earlier "dictionary changed size during iteration" issue.
# ============================================================

namespace_snapshot_628hc1a = list(
    globals().items()
)

print(
    "\n--- NAMESPACE SNAPSHOT ---"
)

print(
    "Objects captured:",
    len(namespace_snapshot_628hc1a)
)


# ============================================================
# 3. FIND CORRUPTION-RELATED CALLABLE HELPERS
# ============================================================

callable_rows_628hc1a = []

keywords_628hc1a = [
    "corrupt",
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
    "scenario",
    "bridge",
    "predict",
    "window",
]


for name_628hc1a, obj_628hc1a in namespace_snapshot_628hc1a:

    if not callable(obj_628hc1a):
        continue

    lname_628hc1a = name_628hc1a.lower()

    if not any(
        key_628hc1a in lname_628hc1a
        for key_628hc1a in keywords_628hc1a
    ):
        continue

    try:
        signature_628hc1a = str(
            inspect.signature(obj_628hc1a)
        )
    except Exception:
        signature_628hc1a = "<signature unavailable>"

    callable_rows_628hc1a.append(
        {
            "function_name": name_628hc1a,
            "signature": signature_628hc1a,
        }
    )


callables_628hc1a = pd.DataFrame(
    callable_rows_628hc1a
).sort_values(
    "function_name"
).reset_index(
    drop=True
)


print(
    "\n--- RELEVANT CALLABLE HELPERS ---"
)

display(
    callables_628hc1a
)


# ============================================================
# 4. FIND CORRUPTED INPUT / WINDOW OBJECTS
# ============================================================

object_rows_628hc1a = []


for name_628hc1a, obj_628hc1a in namespace_snapshot_628hc1a:

    lname_628hc1a = name_628hc1a.lower()

    if not any(
        key_628hc1a in lname_628hc1a
        for key_628hc1a in [
            "corrupt",
            "scenario",
            "628h",
            "628g",
        ]
    ):
        continue

    obj_type_628hc1a = type(
        obj_628hc1a
    ).__name__

    shape_628hc1a = getattr(
        obj_628hc1a,
        "shape",
        None
    )

    try:
        length_628hc1a = len(
            obj_628hc1a
        )
    except Exception:
        length_628hc1a = None

    object_rows_628hc1a.append(
        {
            "object_name": name_628hc1a,
            "type": obj_type_628hc1a,
            "shape": shape_628hc1a,
            "length": length_628hc1a,
        }
    )


objects_628hc1a = pd.DataFrame(
    object_rows_628hc1a
).sort_values(
    "object_name"
).reset_index(
    drop=True
)


print(
    "\n--- CORRUPTION / BRIDGE OBJECT CANDIDATES ---"
)

display(
    objects_628hc1a
)


# ============================================================
# 5. MANIFEST CONTRACT
# ============================================================

manifest_628hc1a = (
    FD002_CORRUPTION_MANIFEST_628GD
)


print(
    "\n--- FROZEN MANIFEST CONTRACT ---"
)

print(
    "Type:",
    type(manifest_628hc1a).__name__
)

print(
    "Shape:",
    getattr(
        manifest_628hc1a,
        "shape",
        None
    )
)


if isinstance(
    manifest_628hc1a,
    pd.DataFrame
):

    print(
        "Columns:",
        list(
            manifest_628hc1a.columns
        )
    )

    print(
        "\nFirst scenario:"
    )

    display(
        manifest_628hc1a.head(1)
    )


# ============================================================
# 6. STORED CORRUPTED-PREDICTION CONTRACT
# ============================================================

stored_corrupt_pred_628hc1a = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB
)


print(
    "\n--- STORED CORRUPTED PREDICTION CONTRACT ---"
)

print(
    "Shape :",
    stored_corrupt_pred_628hc1a.shape
)

print(
    "Finite:",
    bool(
        np.isfinite(
            stored_corrupt_pred_628hc1a
        ).all()
    )
)


if stored_corrupt_pred_628hc1a.shape != (
    315,
    8367
):
    raise RuntimeError(
        "Unexpected corrupted-prediction matrix shape."
    )


# ============================================================
# 7. MODEL / MC CONTRACT
# ============================================================

model_628hc1a = MODEL_FD002_628EB


try:
    model_device_628hc1a = next(
        model_628hc1a.parameters()
    ).device
except Exception:
    model_device_628hc1a = "<unavailable>"


print(
    "\n--- MODEL / MC CONTRACT ---"
)

print(
    "Model class :",
    type(model_628hc1a).__name__
)

print(
    "Device      :",
    model_device_628hc1a
)

print(
    "MC passes   : 50"
)

print(
    "MC STD ddof : 0"
)


# ============================================================
# 8. SAFETY CONTRACT
# ============================================================

FD002_CORRUPTED_MC_PREFLIGHT_628HC1A = {
    "manifest_scenarios": 315,
    "dev_windows": 8367,
    "mc_passes": 50,
    "mc_std_ddof": 0,
    "clean_reliability_chain_closed": True,
    "official_test_labels_read": False,
    "test_scored": False,
    "threshold_selected": False,
    "model_retrained": False,
    "files_written": False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Frozen corruption scenarios : 315"
)

print(
    "DEV windows                 : 8367"
)

print(
    "Stored prediction matrix    :",
    stored_corrupt_pred_628hc1a.shape
)

print(
    "Clean reliability chain     : CLOSED"
)

print(
    "MC passes planned           : 50"
)

print(
    "MC STD convention           : ddof=0"
)

print(
    "Official TEST labels read   : False"
)

print(
    "TEST scored                 : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Model retrained             : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.28H-C1-A ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C1-B
# SINGLE-SCENARIO CORRUPTED MC-DROPOUT SMOKE + TIMING
#
# PURPOSE
# -------
# Validate the exact frozen corrupted-data -> model bridge for
# one FD002 DEV corruption scenario, then execute the frozen
# 50-pass MC-dropout procedure and record its runtime/output
# structure before scaling to all 315 scenarios.
#
# IMPORTANT
# ---------
# - DEV only
# - No official TEST labels
# - No threshold selection
# - No model retraining
# - No files written
# ============================================================

import time
import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-B — SINGLE-SCENARIO CORRUPTED MC SMOKE\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628hc1b = [
    "MODEL_FD002_628EB",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
    "predict_gru_628hc0",
    "mc_dropout_predict_628hc0",
]


print("\n--- REQUIRED OBJECTS ---")

missing_628hc1b = []

for name_628hc1b in required_628hc1b:

    exists_628hc1b = (
        name_628hc1b in globals()
    )

    print(
        f"{name_628hc1b:<55}: "
        f"{exists_628hc1b}"
    )

    if not exists_628hc1b:
        missing_628hc1b.append(
            name_628hc1b
        )


if missing_628hc1b:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_628hc1b)
    )


# ============================================================
# 2. SELECT THE FIRST FROZEN SCENARIO
# ============================================================

manifest_628hc1b = (
    FD002_CORRUPTION_MANIFEST_628GD
)

scenario_index_628hc1b = 0

scenario_628hc1b = (
    manifest_628hc1b
    .iloc[scenario_index_628hc1b]
    .copy()
)


scenario_id_628hc1b = str(
    scenario_628hc1b["scenario_id"]
)

family_628hc1b = str(
    scenario_628hc1b["family"]
)

sensor_628hc1b = str(
    scenario_628hc1b["sensor"]
)

severity_628hc1b = str(
    scenario_628hc1b["severity"]
)


print(
    "\n--- SELECTED FROZEN SCENARIO ---"
)

print(
    "Manifest index :",
    scenario_index_628hc1b
)

print(
    "Scenario ID    :",
    scenario_id_628hc1b
)

print(
    "Family         :",
    family_628hc1b
)

print(
    "Sensor         :",
    sensor_628hc1b
)

print(
    "Severity       :",
    severity_628hc1b
)

print(
    "Alpha          :",
    scenario_628hc1b.get(
        "alpha",
        np.nan
    )
)

print(
    "Direction      :",
    scenario_628hc1b.get(
        "direction",
        np.nan
    )
)

print(
    "Fraction       :",
    scenario_628hc1b.get(
        "requested_fraction",
        np.nan
    )
)

print(
    "Seed           :",
    scenario_628hc1b.get(
        "seed",
        np.nan
    )
)


# ============================================================
# 3. PROTECT THE CLEAN SOURCE
# ============================================================

base_raw_628hc1b = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .copy(deep=True)
)

base_raw_before_628hc1b = (
    base_raw_628hc1b
    .copy(deep=True)
)


# ============================================================
# 4. APPLY THE EXACT FROZEN CORRUPTION
# ============================================================

t0_628hc1b = time.perf_counter()

corrupted_raw_628hc1b = (
    apply_scenario_628hc0(
        base_raw_628hc1b,
        scenario_628hc1b
    )
)

corruption_runtime_628hc1b = (
    time.perf_counter()
    - t0_628hc1b
)


if not isinstance(
    corrupted_raw_628hc1b,
    pd.DataFrame
):
    raise RuntimeError(
        "Corruption helper did not return "
        "a pandas DataFrame."
    )


print(
    "\n--- CORRUPTION APPLICATION ---"
)

print(
    "Output shape :",
    corrupted_raw_628hc1b.shape
)

print(
    "Runtime (s)  :",
    corruption_runtime_628hc1b
)


# ============================================================
# 5. VERIFY CLEAN SOURCE WAS NOT MUTATED
# ============================================================

clean_source_unchanged_628hc1b = (
    base_raw_628hc1b.equals(
        base_raw_before_628hc1b
    )
)


print(
    "Clean source unchanged:",
    clean_source_unchanged_628hc1b
)


if not clean_source_unchanged_628hc1b:
    raise RuntimeError(
        "The clean DEV corruption source was mutated."
    )


# ============================================================
# 6. NORMALIZE THE CORRUPTED SENSOR
# ============================================================

t0_628hc1b = time.perf_counter()

normalized_corrupted_628hc1b = (
    normalize_corrupted_sensor_628ha(
        corrupted_raw_628hc1b,
        sensor_628hc1b
    )
)

normalization_runtime_628hc1b = (
    time.perf_counter()
    - t0_628hc1b
)


if not isinstance(
    normalized_corrupted_628hc1b,
    pd.DataFrame
):
    raise RuntimeError(
        "Normalization helper did not return "
        "a pandas DataFrame."
    )


print(
    "\n--- CORRUPTED NORMALIZATION ---"
)

print(
    "Normalized rows:",
    len(normalized_corrupted_628hc1b)
)

print(
    "Runtime (s)  :",
    normalization_runtime_628hc1b
)


# ============================================================
# 7. BUILD THE EXACT DEV WINDOWS
# ============================================================

t0_628hc1b = time.perf_counter()

window_result_628hc1b = (
    build_windows_628hc0(
        normalized_corrupted_628hc1b
    )
)

window_runtime_628hc1b = (
    time.perf_counter()
    - t0_628hc1b
)


# ------------------------------------------------------------
# Resolve the actual X array safely.
# The frozen helper is expected to return the X array directly,
# but this resolver also protects us if the helper wraps it.
# ------------------------------------------------------------

if isinstance(
    window_result_628hc1b,
    np.ndarray
):

    X_corrupted_628hc1b = (
        window_result_628hc1b
    )

elif isinstance(
    window_result_628hc1b,
    (tuple, list)
):

    array_candidates_628hc1b = [
        np.asarray(obj_628hc1b)
        for obj_628hc1b
        in window_result_628hc1b
        if isinstance(
            obj_628hc1b,
            np.ndarray
        )
        and np.asarray(
            obj_628hc1b
        ).ndim == 3
    ]

    if len(
        array_candidates_628hc1b
    ) != 1:
        raise RuntimeError(
            "Could not uniquely resolve the "
            "3-D window array."
        )

    X_corrupted_628hc1b = (
        array_candidates_628hc1b[0]
    )

else:

    raise RuntimeError(
        "Unexpected build_windows_628hc0 "
        "return type: "
        + type(
            window_result_628hc1b
        ).__name__
    )


X_corrupted_628hc1b = np.asarray(
    X_corrupted_628hc1b
)


print(
    "\n--- CORRUPTED WINDOW CONTRACT ---"
)

print(
    "Shape       :",
    X_corrupted_628hc1b.shape
)

print(
    "Finite      :",
    bool(
        np.isfinite(
            X_corrupted_628hc1b
        ).all()
    )
)

print(
    "Runtime (s) :",
    window_runtime_628hc1b
)


if X_corrupted_628hc1b.shape != (
    8367,
    30,
    17
):
    raise RuntimeError(
        "Unexpected corrupted DEV window shape."
    )


if not np.isfinite(
    X_corrupted_628hc1b
).all():
    raise RuntimeError(
        "Corrupted DEV windows contain "
        "non-finite values."
    )


# ============================================================
# 8. DETERMINISTIC PREDICTION REPRODUCTION
#
# This MUST agree with the already-frozen H-B prediction
# matrix for scenario 001 before we trust the MC stage.
# ============================================================

t0_628hc1b = time.perf_counter()

det_pred_628hc1b = np.asarray(
    predict_gru_628hc0(
        X_corrupted_628hc1b,
        batch_size=256
    )
).reshape(-1)

det_runtime_628hc1b = (
    time.perf_counter()
    - t0_628hc1b
)


stored_det_pred_628hc1b = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB[
        scenario_index_628hc1b
    ]
).reshape(-1)


if len(
    det_pred_628hc1b
) != 8367:

    raise RuntimeError(
        "Unexpected deterministic prediction count."
    )


if len(
    stored_det_pred_628hc1b
) != 8367:

    raise RuntimeError(
        "Unexpected stored prediction count."
    )


det_abs_difference_628hc1b = np.abs(
    det_pred_628hc1b
    - stored_det_pred_628hc1b
)


det_max_error_628hc1b = float(
    np.max(
        det_abs_difference_628hc1b
    )
)

det_mean_error_628hc1b = float(
    np.mean(
        det_abs_difference_628hc1b
    )
)


det_reproduced_628hc1b = bool(
    det_max_error_628hc1b <= 1e-5
)


print(
    "\n--- DETERMINISTIC CORRUPTED-PREDICTION REPRODUCTION ---"
)

print(
    "Stored count             :",
    len(stored_det_pred_628hc1b)
)

print(
    "Reproduced count         :",
    len(det_pred_628hc1b)
)

print(
    "Maximum absolute error   :",
    det_max_error_628hc1b
)

print(
    "Mean absolute error      :",
    det_mean_error_628hc1b
)

print(
    "Reproduced <= 1e-5       :",
    det_reproduced_628hc1b
)

print(
    "Runtime (s)              :",
    det_runtime_628hc1b
)


if not det_reproduced_628hc1b:

    raise RuntimeError(
        "Stored corrupted deterministic "
        "predictions were not reproduced. "
        "Do NOT run corrupted MC-dropout."
    )


# ============================================================
# 9. RUN THE FROZEN MC-DROPOUT PROCEDURE
# ============================================================

print(
    "\n--- CORRUPTED MC-DROPOUT ---"
)

print(
    "Running 50 stochastic forward passes..."
)


t0_628hc1b = time.perf_counter()

mc_result_628hc1b = (
    mc_dropout_predict_628hc0(
        X_corrupted_628hc1b,
        MODEL_FD002_628EB,
        passes=50,
        batch_size=512,
        seed=42
    )
)

mc_runtime_628hc1b = (
    time.perf_counter()
    - t0_628hc1b
)


print(
    "Completed."
)

print(
    "Runtime (s):",
    mc_runtime_628hc1b
)


# ============================================================
# 10. INSPECT MC RETURN STRUCTURE SAFELY
# ============================================================

print(
    "\n--- MC RETURN STRUCTURE ---"
)

print(
    "Top-level type:",
    type(
        mc_result_628hc1b
    ).__name__
)


def describe_mc_item_628hc1b(
    label_628hc1b,
    obj_628hc1b
):

    obj_type_628hc1b = type(
        obj_628hc1b
    ).__name__

    shape_628hc1b = getattr(
        obj_628hc1b,
        "shape",
        None
    )

    try:
        arr_628hc1b = np.asarray(
            obj_628hc1b
        )

        finite_628hc1b = (
            bool(
                np.isfinite(
                    arr_628hc1b
                ).all()
            )
            if np.issubdtype(
                arr_628hc1b.dtype,
                np.number
            )
            else None
        )

    except Exception:
        finite_628hc1b = None

    print(
        f"{label_628hc1b:<20}"
        f"type={obj_type_628hc1b:<15}"
        f" shape={str(shape_628hc1b):<20}"
        f" finite={finite_628hc1b}"
    )


if isinstance(
    mc_result_628hc1b,
    dict
):

    print(
        "Dictionary keys:",
        list(
            mc_result_628hc1b.keys()
        )
    )

    for (
        key_628hc1b,
        value_628hc1b
    ) in mc_result_628hc1b.items():

        describe_mc_item_628hc1b(
            str(key_628hc1b),
            value_628hc1b
        )


elif isinstance(
    mc_result_628hc1b,
    (tuple, list)
):

    print(
        "Elements:",
        len(
            mc_result_628hc1b
        )
    )

    for (
        idx_628hc1b,
        value_628hc1b
    ) in enumerate(
        mc_result_628hc1b
    ):

        describe_mc_item_628hc1b(
            f"element_{idx_628hc1b}",
            value_628hc1b
        )


else:

    describe_mc_item_628hc1b(
        "mc_result",
        mc_result_628hc1b
    )


# ============================================================
# 11. RUNTIME PROJECTION
#
# This is only a computational estimate.
# No methodological decision is made from it.
# ============================================================

projected_seconds_628hc1b = (
    mc_runtime_628hc1b
    * 315.0
)

projected_minutes_628hc1b = (
    projected_seconds_628hc1b
    / 60.0
)


print(
    "\n--- COMPUTE PROJECTION ---"
)

print(
    "One-scenario MC runtime (s) :",
    mc_runtime_628hc1b
)

print(
    "Naive 315-scenario estimate :",
    projected_minutes_628hc1b,
    "minutes"
)


# ============================================================
# 12. FREEZE THIS SMOKE-TEST EVIDENCE IN MEMORY
# ============================================================

FD002_CORRUPTED_MC_SMOKE_628HC1B = {
    "scenario_index": int(
        scenario_index_628hc1b
    ),
    "scenario_id": scenario_id_628hc1b,
    "family": family_628hc1b,
    "sensor": sensor_628hc1b,
    "severity": severity_628hc1b,
    "window_shape": tuple(
        X_corrupted_628hc1b.shape
    ),
    "deterministic_max_abs_error":
        det_max_error_628hc1b,
    "deterministic_mean_abs_error":
        det_mean_error_628hc1b,
    "deterministic_reproduced":
        det_reproduced_628hc1b,
    "corruption_runtime_sec":
        float(
            corruption_runtime_628hc1b
        ),
    "normalization_runtime_sec":
        float(
            normalization_runtime_628hc1b
        ),
    "window_runtime_sec":
        float(
            window_runtime_628hc1b
        ),
    "deterministic_runtime_sec":
        float(
            det_runtime_628hc1b
        ),
    "mc_runtime_sec":
        float(
            mc_runtime_628hc1b
        ),
    "projected_315_mc_minutes":
        float(
            projected_minutes_628hc1b
        ),
    "mc_passes": 50,
    "mc_batch_size": 512,
    "mc_seed": 42,
    "mc_std_ddof": 0,
    "official_test_labels_read": False,
    "test_scored": False,
    "threshold_selected": False,
    "model_retrained": False,
    "files_written": False,
}


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-B FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Scenario                  :",
    scenario_id_628hc1b
)

print(
    "Family / sensor / severity:",
    family_628hc1b,
    "/",
    sensor_628hc1b,
    "/",
    severity_628hc1b
)

print(
    "Corrupted window shape    :",
    X_corrupted_628hc1b.shape
)

print(
    "Windows finite            :",
    bool(
        np.isfinite(
            X_corrupted_628hc1b
        ).all()
    )
)

print(
    "Deterministic reproduced  :",
    det_reproduced_628hc1b
)

print(
    "Deterministic max error   :",
    det_max_error_628hc1b
)

print(
    "MC passes                 : 50"
)

print(
    "MC runtime (sec)          :",
    mc_runtime_628hc1b
)

print(
    "Projected 315 runtime min :",
    projected_minutes_628hc1b
)

print(
    "Official TEST labels read : False"
)

print(
    "TEST scored               : False"
)

print(
    "Threshold selected        : False"
)

print(
    "Model retrained           : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.28H-C1-B ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C1-B-R1
# MC-DROPOUT RETURN-SEMANTICS AUDIT
#
# PURPOSE
# -------
# Determine and freeze the exact meaning/order of the
# 3 values returned by mc_dropout_predict_628hc0.
#
# No model execution.
# No MC rerun.
# No TEST labels.
# No files written.
# ============================================================

import numpy as np


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-B-R1 — MC RETURN-SEMANTICS AUDIT\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required smoke-test objects
# ------------------------------------------------------------

required_628hc1br1 = [
    "mc_result_628hc1b",
    "det_pred_628hc1b",
    "mc_runtime_628hc1b",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc1br1 = []

for name_628hc1br1 in required_628hc1br1:

    exists_628hc1br1 = (
        name_628hc1br1 in globals()
    )

    print(
        f"{name_628hc1br1:<30}: "
        f"{exists_628hc1br1}"
    )

    if not exists_628hc1br1:
        missing_628hc1br1.append(
            name_628hc1br1
        )


if missing_628hc1br1:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_628hc1br1)
    )


# ------------------------------------------------------------
# 2. Structural contract
# ------------------------------------------------------------

if not isinstance(
    mc_result_628hc1b,
    (tuple, list)
):
    raise RuntimeError(
        "Expected MC result to be tuple/list."
    )


if len(mc_result_628hc1b) != 3:
    raise RuntimeError(
        "Expected exactly three MC return values."
    )


candidate_0_628hc1br1 = np.asarray(
    mc_result_628hc1b[0],
    dtype=np.float64
).reshape(-1)

candidate_1_628hc1br1 = np.asarray(
    mc_result_628hc1b[1],
    dtype=np.float64
).reshape(-1)

candidate_2_628hc1br1 = float(
    mc_result_628hc1b[2]
)

det_628hc1br1 = np.asarray(
    det_pred_628hc1b,
    dtype=np.float64
).reshape(-1)


if candidate_0_628hc1br1.shape != (8367,):
    raise RuntimeError(
        "Unexpected element_0 shape."
    )

if candidate_1_628hc1br1.shape != (8367,):
    raise RuntimeError(
        "Unexpected element_1 shape."
    )


# ------------------------------------------------------------
# 3. Descriptive audit
# ------------------------------------------------------------

def summary_628hc1br1(
    name,
    values
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    return {
        "object": name,
        "n": int(values.size),
        "minimum": float(np.min(values)),
        "q25": float(np.quantile(values, 0.25)),
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "q75": float(np.quantile(values, 0.75)),
        "maximum": float(np.max(values)),
        "finite": bool(np.isfinite(values).all()),
        "all_nonnegative": bool(
            np.all(values >= 0.0)
        ),
        "mean_abs_difference_from_deterministic":
            float(
                np.mean(
                    np.abs(
                        values
                        - det_628hc1br1
                    )
                )
            ),
    }


summary_0_628hc1br1 = (
    summary_628hc1br1(
        "element_0",
        candidate_0_628hc1br1
    )
)

summary_1_628hc1br1 = (
    summary_628hc1br1(
        "element_1",
        candidate_1_628hc1br1
    )
)


print("\n--- ELEMENT 0 ---")
for key, value in summary_0_628hc1br1.items():
    print(
        f"{key:<42}: {value}"
    )


print("\n--- ELEMENT 1 ---")
for key, value in summary_1_628hc1br1.items():
    print(
        f"{key:<42}: {value}"
    )


print("\n--- ELEMENT 2 ---")
print(
    "Returned scalar       :",
    candidate_2_628hc1br1
)

print(
    "Externally timed MC run:",
    mc_runtime_628hc1b
)


# ------------------------------------------------------------
# 4. Resolve mean vs standard deviation
#
# MC predictive mean should remain near the corresponding
# deterministic model prediction.
#
# The other vector must satisfy the standard-deviation
# non-negativity contract.
# ------------------------------------------------------------

distance_0_628hc1br1 = float(
    np.mean(
        np.abs(
            candidate_0_628hc1br1
            - det_628hc1br1
        )
    )
)

distance_1_628hc1br1 = float(
    np.mean(
        np.abs(
            candidate_1_628hc1br1
            - det_628hc1br1
        )
    )
)


if distance_0_628hc1br1 < distance_1_628hc1br1:

    mc_mean_628hc1br1 = (
        candidate_0_628hc1br1
    )

    mc_std_628hc1br1 = (
        candidate_1_628hc1br1
    )

    mean_position_628hc1br1 = 0
    std_position_628hc1br1 = 1

else:

    mc_mean_628hc1br1 = (
        candidate_1_628hc1br1
    )

    mc_std_628hc1br1 = (
        candidate_0_628hc1br1
    )

    mean_position_628hc1br1 = 1
    std_position_628hc1br1 = 0


# ------------------------------------------------------------
# 5. Validate resolved uncertainty
# ------------------------------------------------------------

mc_mean_finite_628hc1br1 = bool(
    np.isfinite(
        mc_mean_628hc1br1
    ).all()
)

mc_std_finite_628hc1br1 = bool(
    np.isfinite(
        mc_std_628hc1br1
    ).all()
)

mc_std_nonnegative_628hc1br1 = bool(
    np.all(
        mc_std_628hc1br1 >= 0.0
    )
)


if not mc_mean_finite_628hc1br1:
    raise RuntimeError(
        "Resolved MC mean contains "
        "non-finite values."
    )

if not mc_std_finite_628hc1br1:
    raise RuntimeError(
        "Resolved MC STD contains "
        "non-finite values."
    )

if not mc_std_nonnegative_628hc1br1:
    raise RuntimeError(
        "Resolved MC STD contains "
        "negative values."
    )


# ------------------------------------------------------------
# 6. Freeze return contract
# ------------------------------------------------------------

FD002_MC_RETURN_CONTRACT_628HC1BR1 = {
    "function":
        "mc_dropout_predict_628hc0",
    "return_type":
        type(mc_result_628hc1b).__name__,
    "return_length":
        len(mc_result_628hc1b),
    "mc_mean_position":
        int(mean_position_628hc1br1),
    "mc_std_position":
        int(std_position_628hc1br1),
    "scalar_position":
        2,
    "mean_shape":
        tuple(mc_mean_628hc1br1.shape),
    "std_shape":
        tuple(mc_std_628hc1br1.shape),
    "mean_finite":
        mc_mean_finite_628hc1br1,
    "std_finite":
        mc_std_finite_628hc1br1,
    "std_nonnegative":
        mc_std_nonnegative_628hc1br1,
    "official_test_labels_read":
        False,
    "model_executed":
        False,
    "mc_rerun":
        False,
    "files_written":
        False,
}


MC_MEAN_SMOKE_FD002_628HC1BR1 = (
    mc_mean_628hc1br1.copy()
)

MC_STD_SMOKE_FD002_628HC1BR1 = (
    mc_std_628hc1br1.copy()
)


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-B-R1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "MC mean tuple position :",
    mean_position_628hc1br1
)

print(
    "MC STD tuple position  :",
    std_position_628hc1br1
)

print(
    "Scalar tuple position  : 2"
)

print(
    "MC mean shape          :",
    mc_mean_628hc1br1.shape
)

print(
    "MC STD shape           :",
    mc_std_628hc1br1.shape
)

print(
    "MC mean finite         :",
    mc_mean_finite_628hc1br1
)

print(
    "MC STD finite          :",
    mc_std_finite_628hc1br1
)

print(
    "MC STD non-negative    :",
    mc_std_nonnegative_628hc1br1
)

print(
    "Mean |MC mean - deterministic|:",
    float(
        np.mean(
            np.abs(
                mc_mean_628hc1br1
                - det_628hc1br1
            )
        )
    )
)

print(
    "MC STD minimum         :",
    float(
        np.min(
            mc_std_628hc1br1
        )
    )
)

print(
    "MC STD median          :",
    float(
        np.median(
            mc_std_628hc1br1
        )
    )
)

print(
    "MC STD maximum         :",
    float(
        np.max(
            mc_std_628hc1br1
        )
    )
)

print(
    "Model executed         : False"
)

print(
    "MC rerun               : False"
)

print(
    "Official TEST labels   : False"
)

print(
    "Files written          : False"
)

print(
    "\n=== END STEP 6.28H-C1-B-R1 ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C1-C
# FULL FD002 CORRUPTED MC-DROPOUT UNCERTAINTY EXPERIMENT
#
# PURPOSE
# -------
# Run the frozen 50-pass MC-dropout procedure for every one of
# the 315 frozen FD002 corruption scenarios.
#
# For each scenario:
#   raw clean DEV
#       -> frozen corruption operator
#       -> frozen condition-aware normalization
#       -> frozen 30-cycle windows
#       -> frozen GRU
#       -> 50-pass MC-dropout
#
# Outputs retained:
#   315 x 8367 MC predictive means
#   315 x 8367 MC predictive standard deviations
#   scenario-level uncertainty summaries
#
# IMPORTANT
# ---------
# - Same frozen GRU.
# - Same 315 corruption scenarios.
# - Same MC seed = 42 for every scenario.
# - No official TEST labels.
# - No TEST scoring.
# - No threshold selection.
# - No files written in this step.
# ============================================================

import time
import numpy as np
import pandas as pd
import torch


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-C — FULL FD002 CORRUPTED MC-DROPOUT\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628hc1c = [
    "MODEL_FD002_628EB",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
    "mc_dropout_predict_628hc0",
    "FD002_MC_RETURN_CONTRACT_628HC1BR1",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc1c = []

for name_628hc1c in required_628hc1c:

    exists_628hc1c = (
        name_628hc1c in globals()
    )

    print(
        f"{name_628hc1c:<48}: "
        f"{exists_628hc1c}"
    )

    if not exists_628hc1c:
        missing_628hc1c.append(
            name_628hc1c
        )


if missing_628hc1c:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_628hc1c)
    )


# ============================================================
# 2. FREEZE EXPERIMENT CONTRACT
# ============================================================

manifest_628hc1c = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

n_scenarios_628hc1c = len(
    manifest_628hc1c
)

n_windows_628hc1c = 8367

mc_passes_628hc1c = 50
mc_batch_size_628hc1c = 512
mc_seed_628hc1c = 42
mc_std_ddof_628hc1c = 0


if n_scenarios_628hc1c != 315:
    raise RuntimeError(
        "Frozen corruption manifest must contain "
        "exactly 315 scenarios."
    )


stored_det_matrix_628hc1c = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)


if stored_det_matrix_628hc1c.shape != (
    315,
    8367
):
    raise RuntimeError(
        "Stored corrupted deterministic prediction "
        "matrix has unexpected shape: "
        f"{stored_det_matrix_628hc1c.shape}"
    )


if not np.isfinite(
    stored_det_matrix_628hc1c
).all():
    raise RuntimeError(
        "Stored deterministic corrupted predictions "
        "contain non-finite values."
    )


print("\n--- FROZEN EXPERIMENT CONTRACT ---")

print(
    "Corruption scenarios :",
    n_scenarios_628hc1c
)

print(
    "DEV windows          :",
    n_windows_628hc1c
)

print(
    "MC passes/scenario   :",
    mc_passes_628hc1c
)

print(
    "MC batch size        :",
    mc_batch_size_628hc1c
)

print(
    "MC seed              :",
    mc_seed_628hc1c
)

print(
    "MC STD convention    : ddof=0"
)

print(
    "Common-random-number policy:",
    "same seed for every scenario"
)


# ============================================================
# 3. CONFIRM RETURN CONTRACT
# ============================================================

mean_pos_628hc1c = int(
    FD002_MC_RETURN_CONTRACT_628HC1BR1[
        "mc_mean_position"
    ]
)

std_pos_628hc1c = int(
    FD002_MC_RETURN_CONTRACT_628HC1BR1[
        "mc_std_position"
    ]
)

scalar_pos_628hc1c = int(
    FD002_MC_RETURN_CONTRACT_628HC1BR1[
        "scalar_position"
    ]
)


if (
    mean_pos_628hc1c != 0
    or std_pos_628hc1c != 1
    or scalar_pos_628hc1c != 2
):
    raise RuntimeError(
        "Unexpected frozen MC return contract."
    )


print("\n--- MC RETURN CONTRACT ---")

print(
    "MC mean position :",
    mean_pos_628hc1c
)

print(
    "MC STD position  :",
    std_pos_628hc1c
)

print(
    "Runtime position :",
    scalar_pos_628hc1c
)


# ============================================================
# 4. PREALLOCATE OUTPUT MATRICES
# ============================================================

MC_MEAN_MATRIX_FD002_628HC1C = np.empty(
    (
        n_scenarios_628hc1c,
        n_windows_628hc1c
    ),
    dtype=np.float64
)

MC_STD_MATRIX_FD002_628HC1C = np.empty(
    (
        n_scenarios_628hc1c,
        n_windows_628hc1c
    ),
    dtype=np.float64
)


runtime_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)

mean_det_gap_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)

median_std_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)

mean_std_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)

q95_std_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)

max_std_vector_628hc1c = np.empty(
    n_scenarios_628hc1c,
    dtype=np.float64
)


# ============================================================
# 5. FULL 315-SCENARIO RUN
# ============================================================

print(
    "\n--- RUNNING 315 CORRUPTED MC-DROPOUT SCENARIOS ---"
)

experiment_start_628hc1c = (
    time.perf_counter()
)


for idx_628hc1c, row_628hc1c in (
    manifest_628hc1c.iterrows()
):

    scenario_start_628hc1c = (
        time.perf_counter()
    )

    # --------------------------------------------------------
    # A. Apply frozen corruption
    # --------------------------------------------------------

    corrupted_raw_628hc1c = (
        apply_scenario_628hc0(
            FD002_DEV_CORRUPTION_BASE_628GC,
            row_628hc1c
        )
    )


    # --------------------------------------------------------
    # B. Normalize only the corrupted sensor using the exact
    #    frozen FD002 FIT-only reference.
    # --------------------------------------------------------

    corrupted_normalized_628hc1c = (
        normalize_corrupted_sensor_628ha(
            corrupted_raw_628hc1c,
            row_628hc1c["sensor"]
        )
    )


    # --------------------------------------------------------
    # C. Rebuild frozen 30-cycle DEV windows
    # --------------------------------------------------------

    X_corrupted_628hc1c = (
        build_windows_628hc0(
            corrupted_normalized_628hc1c
        )
    )

    X_corrupted_628hc1c = np.asarray(
        X_corrupted_628hc1c,
        dtype=np.float32
    )


    if X_corrupted_628hc1c.shape != (
        8367,
        30,
        17
    ):
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "unexpected corrupted-window shape "
            f"{X_corrupted_628hc1c.shape}"
        )


    if not np.isfinite(
        X_corrupted_628hc1c
    ).all():
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "corrupted windows contain "
            "non-finite values."
        )


    # --------------------------------------------------------
    # D. Frozen 50-pass MC-dropout
    #
    # Same seed intentionally used for every corruption.
    # --------------------------------------------------------

    mc_result_628hc1c = (
        mc_dropout_predict_628hc0(
            X_corrupted_628hc1c,
            MODEL_FD002_628EB,
            passes=mc_passes_628hc1c,
            batch_size=mc_batch_size_628hc1c,
            seed=mc_seed_628hc1c
        )
    )


    if not isinstance(
        mc_result_628hc1c,
        (tuple, list)
    ):
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "MC result is not tuple/list."
        )


    if len(mc_result_628hc1c) != 3:
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "MC result does not contain "
            "three elements."
        )


    mc_mean_628hc1c = np.asarray(
        mc_result_628hc1c[
            mean_pos_628hc1c
        ],
        dtype=np.float64
    ).reshape(-1)


    mc_std_628hc1c = np.asarray(
        mc_result_628hc1c[
            std_pos_628hc1c
        ],
        dtype=np.float64
    ).reshape(-1)


    returned_runtime_628hc1c = float(
        mc_result_628hc1c[
            scalar_pos_628hc1c
        ]
    )


    # --------------------------------------------------------
    # E. Numerical integrity
    # --------------------------------------------------------

    if mc_mean_628hc1c.shape != (
        n_windows_628hc1c,
    ):
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "unexpected MC mean shape."
        )


    if mc_std_628hc1c.shape != (
        n_windows_628hc1c,
    ):
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "unexpected MC STD shape."
        )


    if not np.isfinite(
        mc_mean_628hc1c
    ).all():
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "MC mean contains non-finite values."
        )


    if not np.isfinite(
        mc_std_628hc1c
    ).all():
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "MC STD contains non-finite values."
        )


    if np.any(
        mc_std_628hc1c < 0.0
    ):
        raise RuntimeError(
            f"Scenario {idx_628hc1c}: "
            "negative MC STD detected."
        )


    # --------------------------------------------------------
    # F. Store scenario outputs
    # --------------------------------------------------------

    MC_MEAN_MATRIX_FD002_628HC1C[
        idx_628hc1c,
        :
    ] = mc_mean_628hc1c


    MC_STD_MATRIX_FD002_628HC1C[
        idx_628hc1c,
        :
    ] = mc_std_628hc1c


    runtime_vector_628hc1c[
        idx_628hc1c
    ] = returned_runtime_628hc1c


    deterministic_reference_628hc1c = (
        stored_det_matrix_628hc1c[
            idx_628hc1c,
            :
        ]
    )


    mean_det_gap_vector_628hc1c[
        idx_628hc1c
    ] = float(
        np.mean(
            np.abs(
                mc_mean_628hc1c
                - deterministic_reference_628hc1c
            )
        )
    )


    median_std_vector_628hc1c[
        idx_628hc1c
    ] = float(
        np.median(
            mc_std_628hc1c
        )
    )


    mean_std_vector_628hc1c[
        idx_628hc1c
    ] = float(
        np.mean(
            mc_std_628hc1c
        )
    )


    q95_std_vector_628hc1c[
        idx_628hc1c
    ] = float(
        np.quantile(
            mc_std_628hc1c,
            0.95
        )
    )


    max_std_vector_628hc1c[
        idx_628hc1c
    ] = float(
        np.max(
            mc_std_628hc1c
        )
    )


    # --------------------------------------------------------
    # G. Progress display
    # --------------------------------------------------------

    completed_628hc1c = (
        idx_628hc1c + 1
    )


    if (
        completed_628hc1c == 1
        or completed_628hc1c % 15 == 0
        or completed_628hc1c
        == n_scenarios_628hc1c
    ):

        elapsed_628hc1c = (
            time.perf_counter()
            - experiment_start_628hc1c
        )

        avg_per_scenario_628hc1c = (
            elapsed_628hc1c
            / completed_628hc1c
        )

        remaining_628hc1c = (
            n_scenarios_628hc1c
            - completed_628hc1c
        )

        eta_minutes_628hc1c = (
            avg_per_scenario_628hc1c
            * remaining_628hc1c
            / 60.0
        )

        print(
            f"[{completed_628hc1c:03d}/"
            f"{n_scenarios_628hc1c:03d}] "
            f"{row_628hc1c['scenario_id']} | "
            f"{row_628hc1c['family']:<7} | "
            f"{row_628hc1c['sensor']:<9} | "
            f"{row_628hc1c['severity']:<6} | "
            f"median_STD="
            f"{median_std_vector_628hc1c[idx_628hc1c]:.4f} | "
            f"mean_STD="
            f"{mean_std_vector_628hc1c[idx_628hc1c]:.4f} | "
            f"elapsed="
            f"{elapsed_628hc1c:.1f}s | "
            f"ETA≈"
            f"{eta_minutes_628hc1c:.1f} min"
        )


# ============================================================
# 6. RESTORE MODEL TO EVALUATION MODE
# ============================================================

MODEL_FD002_628EB.eval()


total_runtime_628hc1c = float(
    time.perf_counter()
    - experiment_start_628hc1c
)


# ============================================================
# 7. GLOBAL MATRIX INTEGRITY
# ============================================================

mean_matrix_shape_ok_628hc1c = (
    MC_MEAN_MATRIX_FD002_628HC1C.shape
    == (315, 8367)
)

std_matrix_shape_ok_628hc1c = (
    MC_STD_MATRIX_FD002_628HC1C.shape
    == (315, 8367)
)

mean_matrix_finite_628hc1c = bool(
    np.isfinite(
        MC_MEAN_MATRIX_FD002_628HC1C
    ).all()
)

std_matrix_finite_628hc1c = bool(
    np.isfinite(
        MC_STD_MATRIX_FD002_628HC1C
    ).all()
)

std_matrix_nonnegative_628hc1c = bool(
    np.all(
        MC_STD_MATRIX_FD002_628HC1C
        >= 0.0
    )
)


if not mean_matrix_shape_ok_628hc1c:
    raise RuntimeError(
        "Final MC mean matrix shape incorrect."
    )

if not std_matrix_shape_ok_628hc1c:
    raise RuntimeError(
        "Final MC STD matrix shape incorrect."
    )

if not mean_matrix_finite_628hc1c:
    raise RuntimeError(
        "Final MC mean matrix contains "
        "non-finite values."
    )

if not std_matrix_finite_628hc1c:
    raise RuntimeError(
        "Final MC STD matrix contains "
        "non-finite values."
    )

if not std_matrix_nonnegative_628hc1c:
    raise RuntimeError(
        "Final MC STD matrix contains "
        "negative values."
    )


# ============================================================
# 8. BUILD SCENARIO-LEVEL SUMMARY TABLE
# ============================================================

FD002_CORRUPTED_MC_SUMMARY_628HC1C = (
    manifest_628hc1c.copy()
)


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mc_runtime_sec"
] = runtime_vector_628hc1c


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mean_abs_mc_mean_minus_deterministic"
] = mean_det_gap_vector_628hc1c


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mc_std_median"
] = median_std_vector_628hc1c


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mc_std_mean"
] = mean_std_vector_628hc1c


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mc_std_q95"
] = q95_std_vector_628hc1c


FD002_CORRUPTED_MC_SUMMARY_628HC1C[
    "mc_std_max"
] = max_std_vector_628hc1c


# ============================================================
# 9. FAMILY x SEVERITY UNCERTAINTY SUMMARY
# ============================================================

FD002_MC_FAMILY_SEVERITY_628HC1C = (
    FD002_CORRUPTED_MC_SUMMARY_628HC1C
    .groupby(
        [
            "family",
            "severity"
        ],
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),
        median_scenario_mc_std=(
            "mc_std_median",
            "median"
        ),
        mean_scenario_mc_std=(
            "mc_std_mean",
            "mean"
        ),
        median_q95_mc_std=(
            "mc_std_q95",
            "median"
        ),
        maximum_mc_std=(
            "mc_std_max",
            "max"
        ),
        mean_mc_mean_det_gap=(
            "mean_abs_mc_mean_minus_deterministic",
            "mean"
        ),
    )
)


# Force intended severity ordering for display only
severity_order_628hc1c = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_order_628hc1c = {
    "bias": 0,
    "noise": 1,
    "drift": 2,
    "dropout": 3,
    "stuck": 4,
}


FD002_MC_FAMILY_SEVERITY_628HC1C[
    "_family_order"
] = (
    FD002_MC_FAMILY_SEVERITY_628HC1C[
        "family"
    ].map(
        family_order_628hc1c
    )
)


FD002_MC_FAMILY_SEVERITY_628HC1C[
    "_severity_order"
] = (
    FD002_MC_FAMILY_SEVERITY_628HC1C[
        "severity"
    ].map(
        severity_order_628hc1c
    )
)


FD002_MC_FAMILY_SEVERITY_628HC1C = (
    FD002_MC_FAMILY_SEVERITY_628HC1C
    .sort_values(
        [
            "_family_order",
            "_severity_order"
        ]
    )
    .drop(
        columns=[
            "_family_order",
            "_severity_order"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 10. TOP UNCERTAINTY SCENARIOS
# ============================================================

FD002_TOP15_MC_STD_628HC1C = (
    FD002_CORRUPTED_MC_SUMMARY_628HC1C
    .sort_values(
        "mc_std_mean",
        ascending=False
    )
    .head(15)
    .reset_index(drop=True)
)


# ============================================================
# 11. FROZEN IN-MEMORY CONTRACT
# ============================================================

FD002_CORRUPTED_MC_CONTRACT_628HC1C = {
    "scenarios": int(
        n_scenarios_628hc1c
    ),
    "dev_windows": int(
        n_windows_628hc1c
    ),
    "mc_passes": int(
        mc_passes_628hc1c
    ),
    "mc_batch_size": int(
        mc_batch_size_628hc1c
    ),
    "mc_seed": int(
        mc_seed_628hc1c
    ),
    "mc_std_ddof": int(
        mc_std_ddof_628hc1c
    ),
    "mean_return_position": int(
        mean_pos_628hc1c
    ),
    "std_return_position": int(
        std_pos_628hc1c
    ),
    "runtime_return_position": int(
        scalar_pos_628hc1c
    ),
    "common_random_numbers":
        True,
    "mc_mean_matrix_shape":
        tuple(
            MC_MEAN_MATRIX_FD002_628HC1C.shape
        ),
    "mc_std_matrix_shape":
        tuple(
            MC_STD_MATRIX_FD002_628HC1C.shape
        ),
    "mc_mean_finite":
        mean_matrix_finite_628hc1c,
    "mc_std_finite":
        std_matrix_finite_628hc1c,
    "mc_std_nonnegative":
        std_matrix_nonnegative_628hc1c,
    "official_test_labels_read":
        False,
    "test_scored":
        False,
    "threshold_selected":
        False,
    "model_retrained":
        False,
    "files_written":
        False,
}


# ============================================================
# 12. OUTPUT REVIEW
# ============================================================

print(
    "\n--- GLOBAL MC MATRIX CONTRACT ---"
)

print(
    "MC mean matrix :",
    MC_MEAN_MATRIX_FD002_628HC1C.shape
)

print(
    "MC STD matrix  :",
    MC_STD_MATRIX_FD002_628HC1C.shape
)

print(
    "MC mean finite :",
    mean_matrix_finite_628hc1c
)

print(
    "MC STD finite  :",
    std_matrix_finite_628hc1c
)

print(
    "MC STD >= 0    :",
    std_matrix_nonnegative_628hc1c
)

print(
    "Total runtime  :",
    total_runtime_628hc1c,
    "seconds"
)

print(
    "Total runtime  :",
    total_runtime_628hc1c / 60.0,
    "minutes"
)


print(
    "\n--- FAMILY x SEVERITY UNCERTAINTY RESPONSE ---"
)

display(
    FD002_MC_FAMILY_SEVERITY_628HC1C
)


print(
    "\n--- TOP 15 SCENARIOS BY MEAN MC STD ---"
)

display(
    FD002_TOP15_MC_STD_628HC1C[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "mc_std_median",
            "mc_std_mean",
            "mc_std_q95",
            "mc_std_max",
            "mean_abs_mc_mean_minus_deterministic",
        ]
    ]
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-C FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Frozen scenarios executed :",
    n_scenarios_628hc1c
)

print(
    "MC passes per scenario     :",
    mc_passes_628hc1c
)

print(
    "Total stochastic passes    :",
    n_scenarios_628hc1c
    * mc_passes_628hc1c
)

print(
    "MC mean matrix shape       :",
    MC_MEAN_MATRIX_FD002_628HC1C.shape
)

print(
    "MC STD matrix shape        :",
    MC_STD_MATRIX_FD002_628HC1C.shape
)

print(
    "All MC means finite        :",
    mean_matrix_finite_628hc1c
)

print(
    "All MC STDs finite         :",
    std_matrix_finite_628hc1c
)

print(
    "All MC STDs non-negative   :",
    std_matrix_nonnegative_628hc1c
)

print(
    "Median scenario MC STD     :",
    float(
        np.median(
            mean_std_vector_628hc1c
        )
    )
)

print(
    "Maximum observed MC STD    :",
    float(
        np.max(
            MC_STD_MATRIX_FD002_628HC1C
        )
    )
)

print(
    "Mean MC-mean/deterministic gap:",
    float(
        np.mean(
            mean_det_gap_vector_628hc1c
        )
    )
)

print(
    "Total runtime (min)        :",
    total_runtime_628hc1c / 60.0
)

print(
    "Official TEST labels read  : False"
)

print(
    "TEST scored                : False"
)

print(
    "Threshold selected         : False"
)

print(
    "Model retrained            : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.28H-C1-C ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C1-D
# FD002 CORRUPTED UNCERTAINTY-ONLY SELECTIVE-RISK RESPONSE
#
# PURPOSE
# -------
# Apply the already-frozen clean FD002 reliability chain to all
# 315 corrupted DEV scenarios:
#
# corrupted deterministic prediction
#       +
# corrupted MC-dropout STD
#       ↓
# FIT-calibrated phase adjustment
#       ↓
# G1 clean-DEV empirical calibration
#       ↓
# corrected engine-balanced G2 calibration
#       ↓
# uncertainty-only selective risk
#
# Then evaluate, on DEV labels only, whether high reliability
# risk corresponds to large RUL prediction error.
#
# IMPORTANT
# ---------
# This is NOT the proposed combined sensor-quality gate yet.
# This step establishes the uncertainty-only baseline.
#
# Official TEST labels are NOT read.
# No final threshold is selected.
# No model is retrained.
# No files are written.
# ============================================================

import time
import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-D — CORRUPTED UNCERTAINTY-ONLY SELECTIVE RISK\n"
    "============================================================"
)


# ============================================================
# 1. REQUIRED OBJECTS
# ============================================================

required_628hc1d = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "MC_STD_MATRIX_FD002_628HC1C",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",
    "DEV_MC_STD_FD002_628FAR2",
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",
    "apply_phase_adjustment_628fbr1",
    "apply_g1_right_ecdf_628fc",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc1d = []

for name_628hc1d in required_628hc1d:

    exists_628hc1d = (
        name_628hc1d in globals()
    )

    print(
        f"{name_628hc1d:<50}: "
        f"{exists_628hc1d}"
    )

    if not exists_628hc1d:
        missing_628hc1d.append(
            name_628hc1d
        )


if missing_628hc1d:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_628hc1d)
    )


# ============================================================
# 2. FREEZE BASIC ARRAYS
# ============================================================

manifest_628hc1d = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

corrupted_predictions_628hc1d = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

corrupted_mc_std_628hc1d = np.asarray(
    MC_STD_MATRIX_FD002_628HC1C,
    dtype=np.float64
)

y_dev_628hc1d = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_predictions_628hc1d = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64
).reshape(-1)

clean_mc_std_628hc1d = np.asarray(
    DEV_MC_STD_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)


n_scenarios_628hc1d = len(
    manifest_628hc1d
)

n_windows_628hc1d = len(
    y_dev_628hc1d
)


expected_matrix_shape_628hc1d = (
    315,
    8367
)


if corrupted_predictions_628hc1d.shape != (
    expected_matrix_shape_628hc1d
):
    raise RuntimeError(
        "Unexpected deterministic corruption matrix shape: "
        f"{corrupted_predictions_628hc1d.shape}"
    )


if corrupted_mc_std_628hc1d.shape != (
    expected_matrix_shape_628hc1d
):
    raise RuntimeError(
        "Unexpected MC-STD corruption matrix shape: "
        f"{corrupted_mc_std_628hc1d.shape}"
    )


if y_dev_628hc1d.shape != (8367,):
    raise RuntimeError(
        "Unexpected DEV target shape: "
        f"{y_dev_628hc1d.shape}"
    )


print("\n--- POPULATION CONTRACT ---")

print(
    "Scenarios :",
    n_scenarios_628hc1d
)

print(
    "DEV windows:",
    n_windows_628hc1d
)

print(
    "Prediction matrix:",
    corrupted_predictions_628hc1d.shape
)

print(
    "MC-STD matrix:",
    corrupted_mc_std_628hc1d.shape
)


# ============================================================
# 3. CORRECTED G2 TRANSFORM
#
# Frozen method:
# weighted empirical mid-rank CDF.
#
# Reference values and weights were fitted on clean FD002 DEV
# candidate endpoints using the already-frozen engine-balanced
# weighting scheme.
# ============================================================

g2_reference_values_628hc1d = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

g2_reference_weights_628hc1d = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)


if (
    g2_reference_values_628hc1d.shape
    != g2_reference_weights_628hc1d.shape
):
    raise RuntimeError(
        "G2 reference values and weights have "
        "different lengths."
    )


if not np.isfinite(
    g2_reference_values_628hc1d
).all():
    raise RuntimeError(
        "G2 reference values contain non-finite values."
    )


if not np.isfinite(
    g2_reference_weights_628hc1d
).all():
    raise RuntimeError(
        "G2 reference weights contain non-finite values."
    )


if np.any(
    g2_reference_weights_628hc1d < 0
):
    raise RuntimeError(
        "G2 reference weights contain negative values."
    )


weight_sum_628hc1d = float(
    np.sum(
        g2_reference_weights_628hc1d
    )
)


if weight_sum_628hc1d <= 0:
    raise RuntimeError(
        "G2 reference weight sum is not positive."
    )


# Normalize defensively without changing relative weighting.
g2_reference_weights_628hc1d = (
    g2_reference_weights_628hc1d
    / weight_sum_628hc1d
)


# Ensure reference ordering and weights remain aligned.
sort_order_628hc1d = np.argsort(
    g2_reference_values_628hc1d,
    kind="mergesort"
)

g2_reference_values_sorted_628hc1d = (
    g2_reference_values_628hc1d[
        sort_order_628hc1d
    ]
)

g2_reference_weights_sorted_628hc1d = (
    g2_reference_weights_628hc1d[
        sort_order_628hc1d
    ]
)

g2_cumulative_weights_628hc1d = np.concatenate(
    [
        np.array(
            [0.0],
            dtype=np.float64
        ),
        np.cumsum(
            g2_reference_weights_sorted_628hc1d
        )
    ]
)


def apply_weighted_midrank_g2_628hc1d(values):
    """
    Weighted empirical mid-rank CDF.

    For each query x:
        P(reference < x)
        + 0.5 * P(reference == x)

    This is the corrected G2 mapping already frozen earlier.
    """

    values = np.asarray(
        values,
        dtype=np.float64
    ).reshape(-1)

    left = np.searchsorted(
        g2_reference_values_sorted_628hc1d,
        values,
        side="left"
    )

    right = np.searchsorted(
        g2_reference_values_sorted_628hc1d,
        values,
        side="right"
    )

    mass_less = (
        g2_cumulative_weights_628hc1d[
            left
        ]
    )

    mass_leq = (
        g2_cumulative_weights_628hc1d[
            right
        ]
    )

    mass_equal = (
        mass_leq
        - mass_less
    )

    result = (
        mass_less
        + 0.5 * mass_equal
    )

    return np.clip(
        result,
        0.0,
        1.0
    )


# ============================================================
# 4. CRITICAL CLEAN-CHAIN REPRODUCTION
#
# We do NOT proceed to corrupted scenarios until the complete
# clean reliability chain reproduces the already-frozen objects.
# ============================================================

print(
    "\n--- CLEAN RELIABILITY-CHAIN REPRODUCTION ---"
)


clean_adjusted_rebuilt_628hc1d = np.asarray(
    apply_phase_adjustment_628fbr1(
        clean_predictions_628hc1d,
        clean_mc_std_628hc1d
    ),
    dtype=np.float64
).reshape(-1)


clean_adjusted_stored_628hc1d = np.asarray(
    FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
    dtype=np.float64
).reshape(-1)


adjusted_error_628hc1d = float(
    np.max(
        np.abs(
            clean_adjusted_rebuilt_628hc1d
            - clean_adjusted_stored_628hc1d
        )
    )
)


clean_g1_rebuilt_628hc1d = np.asarray(
    apply_g1_right_ecdf_628fc(
        clean_adjusted_rebuilt_628hc1d
    ),
    dtype=np.float64
).reshape(-1)


clean_g1_stored_628hc1d = np.asarray(
    FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
    dtype=np.float64
).reshape(-1)


g1_error_628hc1d = float(
    np.max(
        np.abs(
            clean_g1_rebuilt_628hc1d
            - clean_g1_stored_628hc1d
        )
    )
)


clean_final_risk_rebuilt_628hc1d = (
    apply_weighted_midrank_g2_628hc1d(
        clean_g1_rebuilt_628hc1d
    )
)


clean_final_risk_stored_628hc1d = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


final_risk_error_628hc1d = float(
    np.max(
        np.abs(
            clean_final_risk_rebuilt_628hc1d
            - clean_final_risk_stored_628hc1d
        )
    )
)


print(
    "Phase-adjustment max error:",
    adjusted_error_628hc1d
)

print(
    "G1 max error              :",
    g1_error_628hc1d
)

print(
    "Final-risk max error      :",
    final_risk_error_628hc1d
)


if adjusted_error_628hc1d > 1e-12:
    raise RuntimeError(
        "Clean phase-adjustment chain was not "
        "exactly reproduced."
    )


if g1_error_628hc1d > 1e-12:
    raise RuntimeError(
        "Clean G1 chain was not exactly reproduced."
    )


if final_risk_error_628hc1d > 1e-12:
    raise RuntimeError(
        "Clean corrected-G2 chain was not "
        "exactly reproduced."
    )


print(
    "Complete clean chain reproduced: True"
)


# ============================================================
# 5. FIXED-COVERAGE DIAGNOSTIC HELPER
#
# This does NOT select an operating threshold.
# It simply asks:
#
# if we retained the lowest-risk X% of predictions,
# what would their MAE be?
#
# Stable sorting provides deterministic tie handling.
# ============================================================

def fixed_coverage_mae_628hc1d(
    abs_error,
    risk,
    coverage
):

    abs_error = np.asarray(
        abs_error,
        dtype=np.float64
    )

    risk = np.asarray(
        risk,
        dtype=np.float64
    )

    n = len(risk)

    k = int(
        np.floor(
            coverage * n
        )
    )

    k = max(
        1,
        min(
            k,
            n
        )
    )

    ordering = np.argsort(
        risk,
        kind="mergesort"
    )

    accepted = ordering[:k]

    return {
        "requested_coverage":
            float(coverage),

        "accepted_count":
            int(k),

        "achieved_coverage":
            float(k / n),

        "mae":
            float(
                np.mean(
                    abs_error[
                        accepted
                    ]
                )
            ),
    }


# ============================================================
# 6. CLEAN UNCERTAINTY-ONLY DIAGNOSTIC
# ============================================================

clean_abs_error_628hc1d = np.abs(
    clean_predictions_628hc1d
    - y_dev_628hc1d
)

clean_mae_628hc1d = float(
    np.mean(
        clean_abs_error_628hc1d
    )
)


clean_spearman_628hc1d = float(
    pd.Series(
        clean_final_risk_rebuilt_628hc1d
    ).corr(
        pd.Series(
            clean_abs_error_628hc1d
        ),
        method="spearman"
    )
)


clean_cov90_628hc1d = (
    fixed_coverage_mae_628hc1d(
        clean_abs_error_628hc1d,
        clean_final_risk_rebuilt_628hc1d,
        0.90
    )
)

clean_cov80_628hc1d = (
    fixed_coverage_mae_628hc1d(
        clean_abs_error_628hc1d,
        clean_final_risk_rebuilt_628hc1d,
        0.80
    )
)

clean_cov70_628hc1d = (
    fixed_coverage_mae_628hc1d(
        clean_abs_error_628hc1d,
        clean_final_risk_rebuilt_628hc1d,
        0.70
    )
)


print(
    "\n--- CLEAN UNCERTAINTY-ONLY DIAGNOSTIC ---"
)

print(
    "Clean MAE:",
    clean_mae_628hc1d
)

print(
    "Risk/error Spearman:",
    clean_spearman_628hc1d
)

print(
    "MAE @ 90% coverage:",
    clean_cov90_628hc1d["mae"]
)

print(
    "MAE @ 80% coverage:",
    clean_cov80_628hc1d["mae"]
)

print(
    "MAE @ 70% coverage:",
    clean_cov70_628hc1d["mae"]
)


# ============================================================
# 7. PREALLOCATE FULL CORRUPTED FINAL-RISK MATRIX
# ============================================================

FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D = (
    np.empty(
        (
            n_scenarios_628hc1d,
            n_windows_628hc1d
        ),
        dtype=np.float64
    )
)


scenario_rows_628hc1d = []


# ============================================================
# 8. APPLY FROZEN RELIABILITY CHAIN TO ALL 315 SCENARIOS
# ============================================================

print(
    "\n--- APPLYING FROZEN RELIABILITY CHAIN "
    "TO 315 CORRUPTED SCENARIOS ---"
)


run_start_628hc1d = (
    time.perf_counter()
)


for i_628hc1d, row_628hc1d in (
    manifest_628hc1d.iterrows()
):

    prediction_628hc1d = (
        corrupted_predictions_628hc1d[
            i_628hc1d,
            :
        ]
    )

    mc_std_628hc1d = (
        corrupted_mc_std_628hc1d[
            i_628hc1d,
            :
        ]
    )


    # --------------------------------------------------------
    # Phase adjustment
    # --------------------------------------------------------

    adjusted_u_628hc1d = np.asarray(
        apply_phase_adjustment_628fbr1(
            prediction_628hc1d,
            mc_std_628hc1d
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # G1 calibration
    # --------------------------------------------------------

    g1_risk_628hc1d = np.asarray(
        apply_g1_right_ecdf_628fc(
            adjusted_u_628hc1d
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # Corrected G2 calibration
    # --------------------------------------------------------

    final_risk_628hc1d = (
        apply_weighted_midrank_g2_628hc1d(
            g1_risk_628hc1d
        )
    )


    if final_risk_628hc1d.shape != (
        n_windows_628hc1d,
    ):
        raise RuntimeError(
            f"Scenario {i_628hc1d}: "
            "unexpected final-risk shape."
        )


    if not np.isfinite(
        final_risk_628hc1d
    ).all():
        raise RuntimeError(
            f"Scenario {i_628hc1d}: "
            "non-finite final risk."
        )


    if (
        np.min(
            final_risk_628hc1d
        ) < 0.0
        or
        np.max(
            final_risk_628hc1d
        ) > 1.0
    ):
        raise RuntimeError(
            f"Scenario {i_628hc1d}: "
            "final risk outside [0,1]."
        )


    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D[
        i_628hc1d,
        :
    ] = final_risk_628hc1d


    # --------------------------------------------------------
    # Prediction error
    # --------------------------------------------------------

    abs_error_628hc1d = np.abs(
        prediction_628hc1d
        - y_dev_628hc1d
    )


    full_mae_628hc1d = float(
        np.mean(
            abs_error_628hc1d
        )
    )


    # --------------------------------------------------------
    # Risk/error ranking association
    # --------------------------------------------------------

    spearman_628hc1d = (
        pd.Series(
            final_risk_628hc1d
        ).corr(
            pd.Series(
                abs_error_628hc1d
            ),
            method="spearman"
        )
    )


    if pd.isna(
        spearman_628hc1d
    ):
        spearman_628hc1d = np.nan
    else:
        spearman_628hc1d = float(
            spearman_628hc1d
        )


    # --------------------------------------------------------
    # Fixed-coverage diagnostics
    # --------------------------------------------------------

    cov90_628hc1d = (
        fixed_coverage_mae_628hc1d(
            abs_error_628hc1d,
            final_risk_628hc1d,
            0.90
        )
    )

    cov80_628hc1d = (
        fixed_coverage_mae_628hc1d(
            abs_error_628hc1d,
            final_risk_628hc1d,
            0.80
        )
    )

    cov70_628hc1d = (
        fixed_coverage_mae_628hc1d(
            abs_error_628hc1d,
            final_risk_628hc1d,
            0.70
        )
    )


    scenario_rows_628hc1d.append(
        {
            "scenario_id":
                row_628hc1d[
                    "scenario_id"
                ],

            "scenario_number":
                int(
                    row_628hc1d[
                        "scenario_number"
                    ]
                ),

            "family":
                row_628hc1d[
                    "family"
                ],

            "sensor":
                row_628hc1d[
                    "sensor"
                ],

            "severity":
                row_628hc1d[
                    "severity"
                ],

            "full_mae":
                full_mae_628hc1d,

            "mean_final_risk":
                float(
                    np.mean(
                        final_risk_628hc1d
                    )
                ),

            "median_final_risk":
                float(
                    np.median(
                        final_risk_628hc1d
                    )
                ),

            "q95_final_risk":
                float(
                    np.quantile(
                        final_risk_628hc1d,
                        0.95
                    )
                ),

            "risk_error_spearman":
                spearman_628hc1d,

            "mae_at_90pct_coverage":
                cov90_628hc1d[
                    "mae"
                ],

            "mae_gain_at_90pct_coverage":
                (
                    full_mae_628hc1d
                    - cov90_628hc1d[
                        "mae"
                    ]
                ),

            "mae_at_80pct_coverage":
                cov80_628hc1d[
                    "mae"
                ],

            "mae_gain_at_80pct_coverage":
                (
                    full_mae_628hc1d
                    - cov80_628hc1d[
                        "mae"
                    ]
                ),

            "mae_at_70pct_coverage":
                cov70_628hc1d[
                    "mae"
                ],

            "mae_gain_at_70pct_coverage":
                (
                    full_mae_628hc1d
                    - cov70_628hc1d[
                        "mae"
                    ]
                ),
        }
    )


    completed_628hc1d = (
        i_628hc1d + 1
    )


    if (
        completed_628hc1d == 1
        or
        completed_628hc1d % 50 == 0
        or
        completed_628hc1d
        == n_scenarios_628hc1d
    ):

        elapsed_628hc1d = (
            time.perf_counter()
            - run_start_628hc1d
        )

        print(
            f"[{completed_628hc1d:03d}/315] "
            f"{row_628hc1d['scenario_id']} | "
            f"{row_628hc1d['family']:<7} | "
            f"{row_628hc1d['severity']:<6} | "
            f"MAE={full_mae_628hc1d:.4f} | "
            f"Spearman="
            f"{spearman_628hc1d:.4f} | "
            f"gain@80="
            f"{full_mae_628hc1d - cov80_628hc1d['mae']:+.4f} | "
            f"elapsed={elapsed_628hc1d:.1f}s"
        )


total_runtime_628hc1d = float(
    time.perf_counter()
    - run_start_628hc1d
)


# ============================================================
# 9. SCENARIO-LEVEL RESPONSE TABLE
# ============================================================

FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D = (
    pd.DataFrame(
        scenario_rows_628hc1d
    )
)


if len(
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D
) != 315:
    raise RuntimeError(
        "Uncertainty-only response table does "
        "not contain 315 scenarios."
    )


# ============================================================
# 10. FAMILY x SEVERITY SUMMARY
# ============================================================

FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D = (
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D
    .groupby(
        [
            "family",
            "severity"
        ],
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        median_full_mae=(
            "full_mae",
            "median"
        ),

        median_risk_error_spearman=(
            "risk_error_spearman",
            "median"
        ),

        mean_risk_error_spearman=(
            "risk_error_spearman",
            "mean"
        ),

        median_gain_90=(
            "mae_gain_at_90pct_coverage",
            "median"
        ),

        median_gain_80=(
            "mae_gain_at_80pct_coverage",
            "median"
        ),

        median_gain_70=(
            "mae_gain_at_70pct_coverage",
            "median"
        ),

        mean_final_risk=(
            "mean_final_risk",
            "mean"
        ),
    )
)


family_order_628hc1d = {
    "bias": 0,
    "noise": 1,
    "drift": 2,
    "dropout": 3,
    "stuck": 4,
}

severity_order_628hc1d = {
    "low": 0,
    "medium": 1,
    "high": 2,
}


FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D[
    "_family_order"
] = (
    FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D[
        "family"
    ].map(
        family_order_628hc1d
    )
)


FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D[
    "_severity_order"
] = (
    FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D[
        "severity"
    ].map(
        severity_order_628hc1d
    )
)


FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D = (
    FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D
    .sort_values(
        [
            "_family_order",
            "_severity_order"
        ]
    )
    .drop(
        columns=[
            "_family_order",
            "_severity_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. WORST UNCERTAINTY-ONLY CASES
#
# Negative gain means abstention made MAE worse.
# These cases are especially informative for the proposed
# sensor-quality + uncertainty method.
# ============================================================

FD002_WORST15_UNCERTAINTY_ONLY_628HC1D = (
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D
    .sort_values(
        "mae_gain_at_80pct_coverage",
        ascending=True
    )
    .head(15)
    .reset_index(drop=True)
)


FD002_BEST15_UNCERTAINTY_ONLY_628HC1D = (
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D
    .sort_values(
        "mae_gain_at_80pct_coverage",
        ascending=False
    )
    .head(15)
    .reset_index(drop=True)
)


# ============================================================
# 12. GLOBAL INTEGRITY
# ============================================================

risk_matrix_shape_ok_628hc1d = (
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D.shape
    == (315, 8367)
)

risk_matrix_finite_628hc1d = bool(
    np.isfinite(
        FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D
    ).all()
)

risk_matrix_range_ok_628hc1d = bool(
    np.all(
        (
            FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D
            >= 0.0
        )
        &
        (
            FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D
            <= 1.0
        )
    )
)


if not risk_matrix_shape_ok_628hc1d:
    raise RuntimeError(
        "Final corrupted reliability-risk matrix "
        "has the wrong shape."
    )

if not risk_matrix_finite_628hc1d:
    raise RuntimeError(
        "Final corrupted reliability-risk matrix "
        "contains non-finite values."
    )

if not risk_matrix_range_ok_628hc1d:
    raise RuntimeError(
        "Final corrupted reliability-risk matrix "
        "contains values outside [0,1]."
    )


# ============================================================
# 13. FROZEN IN-MEMORY CONTRACT
# ============================================================

FD002_UNCERTAINTY_ONLY_CONTRACT_628HC1D = {
    "scenarios": 315,
    "dev_windows": 8367,
    "phase_adjustment":
        "frozen_FD002_FIT_calibration",
    "g1_reference":
        "clean_FD002_DEV_windows",
    "g2_reference":
        "engine_balanced_clean_FD002_DEV_candidate_endpoints",
    "g2_transform":
        "weighted_empirical_midrank_CDF",
    "risk_direction":
        "larger_value_means_lower_reliability",
    "coverage_diagnostic":
        "fixed_count_lowest_risk_first",
    "coverage_levels":
        [0.90, 0.80, 0.70],
    "clean_chain_exact":
        True,
    "risk_matrix_shape":
        tuple(
            FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D.shape
        ),
    "risk_finite":
        risk_matrix_finite_628hc1d,
    "risk_within_0_1":
        risk_matrix_range_ok_628hc1d,
    "official_test_labels_read":
        False,
    "test_scored":
        False,
    "threshold_selected":
        False,
    "model_retrained":
        False,
    "files_written":
        False,
}


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print(
    "\n--- FAMILY x SEVERITY: "
    "UNCERTAINTY-ONLY SELECTIVE RESPONSE ---"
)

display(
    FD002_UNCERTAINTY_ONLY_FAMILY_SEVERITY_628HC1D
)


print(
    "\n--- WORST 15 CASES AT 80% COVERAGE ---"
)

display(
    FD002_WORST15_UNCERTAINTY_ONLY_628HC1D[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "full_mae",
            "risk_error_spearman",
            "mae_at_80pct_coverage",
            "mae_gain_at_80pct_coverage",
            "mean_final_risk",
        ]
    ]
)


print(
    "\n--- BEST 15 CASES AT 80% COVERAGE ---"
)

display(
    FD002_BEST15_UNCERTAINTY_ONLY_628HC1D[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "full_mae",
            "risk_error_spearman",
            "mae_at_80pct_coverage",
            "mae_gain_at_80pct_coverage",
            "mean_final_risk",
        ]
    ]
)


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

positive_gain_80_628hc1d = int(
    (
        FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D[
            "mae_gain_at_80pct_coverage"
        ] > 0
    ).sum()
)

negative_gain_80_628hc1d = int(
    (
        FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D[
            "mae_gain_at_80pct_coverage"
        ] < 0
    ).sum()
)

median_spearman_628hc1d = float(
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D[
        "risk_error_spearman"
    ].median()
)

median_gain80_628hc1d = float(
    FD002_UNCERTAINTY_ONLY_RESPONSE_628HC1D[
        "mae_gain_at_80pct_coverage"
    ].median()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.28H-C1-D FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Scenarios evaluated        :",
    315
)

print(
    "DEV windows/scenario       :",
    8367
)

print(
    "Risk matrix shape          :",
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D.shape
)

print(
    "Risk matrix finite         :",
    risk_matrix_finite_628hc1d
)

print(
    "Risk values within [0,1]   :",
    risk_matrix_range_ok_628hc1d
)

print(
    "Complete clean chain exact : True"
)

print(
    "Median risk/error Spearman :",
    median_spearman_628hc1d
)

print(
    "Median MAE gain @80% cov.  :",
    median_gain80_628hc1d
)

print(
    "Scenarios gain @80% > 0    :",
    positive_gain_80_628hc1d,
    "/ 315"
)

print(
    "Scenarios gain @80% < 0    :",
    negative_gain_80_628hc1d,
    "/ 315"
)

print(
    "Runtime (sec)              :",
    total_runtime_628hc1d
)

print(
    "Official TEST labels read  : False"
)

print(
    "TEST scored                : False"
)

print(
    "Threshold selected         : False"
)

print(
    "Model retrained            : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.28H-C1-D ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C2-A — FD002 LABEL-FREE SENSOR-QUALITY SIGNAL PRE-FLIGHT
# ================================================================
#
# PURPOSE
# -------
# Build and audit LABEL-FREE sensor-quality indicators using:
#
#   - clean FD002 FIT windows as the statistical reference
#   - clean FD002 DEV windows for a clean diagnostic population
#   - a small frozen probe set of corrupted DEV scenarios
#
# IMPORTANT
# ---------
# * NO DEV RUL labels are used.
# * NO official TEST labels are read.
# * NO RUL model is trained/retrained.
# * NO abstention threshold is selected.
# * NO files are written.
#
# This is a diagnostic / methodology pre-flight only.
# ================================================================

import time
import numpy as np
import pandas as pd

print("=" * 68)
print("STEP 6.28H-C2-A — FD002 LABEL-FREE SENSOR-QUALITY PRE-FLIGHT")
print("=" * 68)


# ================================================================
# 1. REQUIRED FROZEN OBJECTS
# ================================================================

required_objects_628hc2a = [
    "X_FIT_FD002_628D",
    "X_DEV_FD002_628D",
    "FD002_MODEL_FEATURES_628C",
    "FD002_MODEL_SENSORS_628GB",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
]

print("\n--- REQUIRED OBJECTS ---")

missing_objects_628hc2a = []

for name in required_objects_628hc2a:
    exists = name in globals()
    print(f"{name:<42}: {exists}")

    if not exists:
        missing_objects_628hc2a.append(name)

if missing_objects_628hc2a:
    raise RuntimeError(
        "Missing required Step 6.28H-C2-A objects:\n"
        + "\n".join(missing_objects_628hc2a)
    )


# ================================================================
# 2. FREEZE BASIC INPUT CONTRACT
# ================================================================

X_fit_628hc2a = np.asarray(X_FIT_FD002_628D)
X_dev_clean_628hc2a = np.asarray(X_DEV_FD002_628D)

model_features_628hc2a = list(FD002_MODEL_FEATURES_628C)
model_sensors_628hc2a = list(FD002_MODEL_SENSORS_628GB)

manifest_628hc2a = FD002_CORRUPTION_MANIFEST_628GD.copy()
dev_corruption_base_628hc2a = FD002_DEV_CORRUPTION_BASE_628GC


print("\n--- INPUT CONTRACT ---")
print("FIT X shape        :", X_fit_628hc2a.shape)
print("Clean DEV X shape  :", X_dev_clean_628hc2a.shape)
print("Model features     :", len(model_features_628hc2a))
print("Model sensors      :", len(model_sensors_628hc2a))
print("Manifest scenarios :", len(manifest_628hc2a))


if X_fit_628hc2a.ndim != 3:
    raise RuntimeError("FIT X is not a 3-D sequence array.")

if X_dev_clean_628hc2a.ndim != 3:
    raise RuntimeError("DEV X is not a 3-D sequence array.")

if X_fit_628hc2a.shape[1:] != (30, 17):
    raise RuntimeError(
        f"Unexpected FIT sequence shape: {X_fit_628hc2a.shape}"
    )

if X_dev_clean_628hc2a.shape != (8367, 30, 17):
    raise RuntimeError(
        f"Unexpected clean DEV sequence shape: {X_dev_clean_628hc2a.shape}"
    )

if len(model_features_628hc2a) != 17:
    raise RuntimeError("Expected exactly 17 model features.")

if len(model_sensors_628hc2a) != 15:
    raise RuntimeError("Expected exactly 15 retained sensors.")

if len(manifest_628hc2a) != 315:
    raise RuntimeError("Expected exactly 315 frozen corruption scenarios.")


# ================================================================
# 3. MAP RAW SENSOR NAMES TO MODEL FEATURE INDICES
# ================================================================

sensor_feature_names_628hc2a = [
    f"{sensor}_z"
    for sensor in model_sensors_628hc2a
]

missing_sensor_features_628hc2a = [
    feature
    for feature in sensor_feature_names_628hc2a
    if feature not in model_features_628hc2a
]

if missing_sensor_features_628hc2a:
    raise RuntimeError(
        "Could not locate normalized sensor features:\n"
        + "\n".join(missing_sensor_features_628hc2a)
    )

sensor_feature_indices_628hc2a = np.asarray(
    [
        model_features_628hc2a.index(feature)
        for feature in sensor_feature_names_628hc2a
    ],
    dtype=int,
)

sensor_to_feature_index_628hc2a = dict(
    zip(
        model_sensors_628hc2a,
        sensor_feature_indices_628hc2a.tolist(),
    )
)


print("\n--- SENSOR FEATURE CONTRACT ---")
print("Retained sensors:", len(model_sensors_628hc2a))

for sensor in model_sensors_628hc2a:
    feature_name = f"{sensor}_z"
    feature_index = sensor_to_feature_index_628hc2a[sensor]

    print(
        f"{sensor:<12} -> "
        f"{feature_name:<14} -> feature index {feature_index}"
    )


# ================================================================
# 4. LABEL-FREE WINDOW STATISTICS
# ================================================================
#
# Each statistic is computed independently for every:
#
#     window × sensor
#
# Statistics:
#
# 1. |window mean|
#       level / offset abnormality
#
# 2. window STD
#       high variability -> possible noise
#       low variability  -> possible stuck sensor
#
# 3. mean absolute first difference
#       temporal roughness
#
# 4. zero fraction
#       useful for detecting mean-imputed dropout after normalization
#
# 5. flat-step fraction
#       consecutive identical values -> stuck behaviour
#
# NONE OF THESE USE RUL LABELS.
# ================================================================

ZERO_TOL_628HC2A = 1e-12
FLAT_TOL_628HC2A = 1e-12


def sensor_window_statistics_628hc2a(X):
    """
    Compute label-free statistics for retained sensor channels.

    Returns a dictionary.
    Every matrix has shape:

        (number_of_windows, 15_sensors)
    """

    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError("X must be a 3-D sequence array.")

    if X.shape[1] != 30:
        raise ValueError(
            f"Expected window length 30, got {X.shape[1]}."
        )

    if X.shape[2] != 17:
        raise ValueError(
            f"Expected 17 model features, got {X.shape[2]}."
        )

    Z = X[:, :, sensor_feature_indices_628hc2a].astype(
        np.float64,
        copy=False,
    )

    if not np.isfinite(Z).all():
        raise RuntimeError(
            "Non-finite values detected in sensor sequence array."
        )

    window_mean = np.mean(
        Z,
        axis=1,
    )

    abs_window_mean = np.abs(window_mean)

    window_std = np.std(
        Z,
        axis=1,
        ddof=0,
    )

    differences = np.diff(
        Z,
        axis=1,
    )

    roughness = np.mean(
        np.abs(differences),
        axis=1,
    )

    zero_fraction = np.mean(
        np.isclose(
            Z,
            0.0,
            atol=ZERO_TOL_628HC2A,
            rtol=0.0,
        ),
        axis=1,
    )

    flat_fraction = np.mean(
        np.isclose(
            differences,
            0.0,
            atol=FLAT_TOL_628HC2A,
            rtol=0.0,
        ),
        axis=1,
    )

    results = {
        "abs_mean": abs_window_mean,
        "std": window_std,
        "roughness": roughness,
        "zero_fraction": zero_fraction,
        "flat_fraction": flat_fraction,
    }

    for name, values in results.items():

        if values.shape != (
            X.shape[0],
            len(model_sensors_628hc2a),
        ):
            raise RuntimeError(
                f"{name} produced unexpected shape {values.shape}."
            )

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{name} contains non-finite values."
            )

    return results


# ================================================================
# 5. BUILD FIT-ONLY STATISTICAL REFERENCE
# ================================================================

start_fit_reference_628hc2a = time.perf_counter()

FIT_SENSOR_STATS_628HC2A = (
    sensor_window_statistics_628hc2a(
        X_fit_628hc2a
    )
)

FIT_SORTED_REFERENCE_628HC2A = {
    name: np.sort(
        values,
        axis=0,
    )
    for name, values
    in FIT_SENSOR_STATS_628HC2A.items()
}

fit_reference_runtime_628hc2a = (
    time.perf_counter()
    - start_fit_reference_628hc2a
)


print("\n--- FIT-ONLY REFERENCE ---")
print("FIT reference windows :", X_fit_628hc2a.shape[0])
print("Sensors               :", len(model_sensors_628hc2a))
print(
    "Reference runtime (s) :",
    fit_reference_runtime_628hc2a,
)


# ================================================================
# 6. EMPIRICAL MID-RANK TAIL TRANSFORMS
# ================================================================
#
# Higher-tail risk:
#
#     large statistic -> risk near 1
#
# Lower-tail risk:
#
#     small statistic -> risk near 1
#
# Ties use the empirical mid-rank convention.
# ================================================================

def empirical_upper_midrank_628hc2a(
    values,
    sorted_reference,
):
    """
    Convert values to empirical percentile scores.

    High values receive scores near 1.

    values:
        shape (N, sensors)

    sorted_reference:
        shape (N_ref, sensors)
    """

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    sorted_reference = np.asarray(
        sorted_reference,
        dtype=np.float64,
    )

    if values.ndim != 2:
        raise ValueError(
            "values must be a 2-D matrix."
        )

    if sorted_reference.ndim != 2:
        raise ValueError(
            "sorted_reference must be a 2-D matrix."
        )

    if values.shape[1] != sorted_reference.shape[1]:
        raise ValueError(
            "Sensor count differs between values and reference."
        )

    n_ref = sorted_reference.shape[0]

    output = np.empty_like(
        values,
        dtype=np.float64,
    )

    for sensor_idx in range(values.shape[1]):

        ref = sorted_reference[:, sensor_idx]
        x = values[:, sensor_idx]

        left = np.searchsorted(
            ref,
            x,
            side="left",
        )

        right = np.searchsorted(
            ref,
            x,
            side="right",
        )

        output[:, sensor_idx] = (
            left + right
        ) / (
            2.0 * n_ref
        )

    return np.clip(
        output,
        0.0,
        1.0,
    )


def empirical_lower_midrank_628hc2a(
    values,
    sorted_reference,
):
    """
    Low values receive scores near 1.
    """

    upper = empirical_upper_midrank_628hc2a(
        values,
        sorted_reference,
    )

    return np.clip(
        1.0 - upper,
        0.0,
        1.0,
    )


# ================================================================
# 7. SENSOR-QUALITY BRANCHES
# ================================================================
#
# Four interpretable branches are created.
#
# LEVEL:
#     bias / drift-like level deviations
#
# NOISE:
#     excessive STD and/or temporal roughness
#
# DROPOUT:
#     unusually large zero/imputed-value fraction
#
# STUCK:
#     unusually low STD,
#     unusually low roughness,
#     or unusually high flat-step fraction
#
# Final combination is NOT frozen here.
# ================================================================

def sensor_quality_branches_628hc2a(X):
    """
    Return window-level label-free sensor-quality risks.

    Output branch arrays each have shape (N_windows,).

    Values lie in [0, 1].
    """

    stats = sensor_window_statistics_628hc2a(
        X
    )

    level_per_sensor = (
        empirical_upper_midrank_628hc2a(
            stats["abs_mean"],
            FIT_SORTED_REFERENCE_628HC2A[
                "abs_mean"
            ],
        )
    )

    high_std_per_sensor = (
        empirical_upper_midrank_628hc2a(
            stats["std"],
            FIT_SORTED_REFERENCE_628HC2A[
                "std"
            ],
        )
    )

    low_std_per_sensor = (
        empirical_lower_midrank_628hc2a(
            stats["std"],
            FIT_SORTED_REFERENCE_628HC2A[
                "std"
            ],
        )
    )

    high_rough_per_sensor = (
        empirical_upper_midrank_628hc2a(
            stats["roughness"],
            FIT_SORTED_REFERENCE_628HC2A[
                "roughness"
            ],
        )
    )

    low_rough_per_sensor = (
        empirical_lower_midrank_628hc2a(
            stats["roughness"],
            FIT_SORTED_REFERENCE_628HC2A[
                "roughness"
            ],
        )
    )

    zero_per_sensor = (
        empirical_upper_midrank_628hc2a(
            stats["zero_fraction"],
            FIT_SORTED_REFERENCE_628HC2A[
                "zero_fraction"
            ],
        )
    )

    flat_per_sensor = (
        empirical_upper_midrank_628hc2a(
            stats["flat_fraction"],
            FIT_SORTED_REFERENCE_628HC2A[
                "flat_fraction"
            ],
        )
    )

    level_branch = np.max(
        level_per_sensor,
        axis=1,
    )

    noise_branch = np.max(
        np.maximum(
            high_std_per_sensor,
            high_rough_per_sensor,
        ),
        axis=1,
    )

    dropout_branch = np.max(
        zero_per_sensor,
        axis=1,
    )

    stuck_branch = np.max(
        np.maximum.reduce(
            [
                low_std_per_sensor,
                low_rough_per_sensor,
                flat_per_sensor,
            ]
        ),
        axis=1,
    )

    overall_raw = np.maximum.reduce(
        [
            level_branch,
            noise_branch,
            dropout_branch,
            stuck_branch,
        ]
    )

    output = {
        "level": level_branch,
        "noise": noise_branch,
        "dropout": dropout_branch,
        "stuck": stuck_branch,
        "overall_raw": overall_raw,
    }

    for name, values in output.items():

        values = np.asarray(values)

        if values.shape != (
            X.shape[0],
        ):
            raise RuntimeError(
                f"{name} branch has unexpected shape "
                f"{values.shape}."
            )

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{name} contains non-finite values."
            )

        if np.min(values) < 0.0:
            raise RuntimeError(
                f"{name} contains values below zero."
            )

        if np.max(values) > 1.0:
            raise RuntimeError(
                f"{name} contains values above one."
            )

    return output


# ================================================================
# 8. FIT AND CLEAN-DEV BRANCH SCORES
# ================================================================

start_clean_quality_628hc2a = time.perf_counter()

FIT_SENSOR_QUALITY_628HC2A = (
    sensor_quality_branches_628hc2a(
        X_fit_628hc2a
    )
)

CLEAN_DEV_SENSOR_QUALITY_628HC2A = (
    sensor_quality_branches_628hc2a(
        X_dev_clean_628hc2a
    )
)

clean_quality_runtime_628hc2a = (
    time.perf_counter()
    - start_clean_quality_628hc2a
)


# ================================================================
# 9. FIT-ONLY Q95 DIAGNOSTIC MARKERS
# ================================================================
#
# These are NOT abstention thresholds.
#
# They are used only to ask:
#
#   "How frequently does a corrupted DEV window exceed a level
#    that is unusually high relative to clean FIT?"
# ================================================================

QUALITY_BRANCH_NAMES_628HC2A = [
    "level",
    "noise",
    "dropout",
    "stuck",
    "overall_raw",
]

FIT_Q95_MARKERS_628HC2A = {
    branch: float(
        np.quantile(
            FIT_SENSOR_QUALITY_628HC2A[
                branch
            ],
            0.95,
            method="linear",
        )
    )
    for branch
    in QUALITY_BRANCH_NAMES_628HC2A
}


print("\n--- FIT Q95 DIAGNOSTIC MARKERS ---")

for branch in QUALITY_BRANCH_NAMES_628HC2A:
    print(
        f"{branch:<12}: "
        f"{FIT_Q95_MARKERS_628HC2A[branch]:.6f}"
    )


# ================================================================
# 10. CLEAN DEV DIAGNOSTIC SUMMARY
# ================================================================

clean_rows_628hc2a = []

for branch in QUALITY_BRANCH_NAMES_628HC2A:

    values = np.asarray(
        CLEAN_DEV_SENSOR_QUALITY_628HC2A[
            branch
        ]
    )

    threshold = (
        FIT_Q95_MARKERS_628HC2A[
            branch
        ]
    )

    clean_rows_628hc2a.append(
        {
            "branch": branch,
            "n": len(values),
            "median": float(
                np.median(values)
            ),
            "mean": float(
                np.mean(values)
            ),
            "q95": float(
                np.quantile(
                    values,
                    0.95,
                    method="linear",
                )
            ),
            "fit_q95_marker": float(
                threshold
            ),
            "fraction_above_fit_q95": float(
                np.mean(
                    values > threshold
                )
            ),
        }
    )

CLEAN_DEV_SENSOR_QUALITY_SUMMARY_628HC2A = (
    pd.DataFrame(
        clean_rows_628hc2a
    )
)


print("\n--- CLEAN DEV SENSOR-QUALITY SUMMARY ---")
display(
    CLEAN_DEV_SENSOR_QUALITY_SUMMARY_628HC2A
)


# ================================================================
# 11. SELECT A SMALL FROZEN PROBE SET
# ================================================================
#
# Use sensor_2 because:
#
#   * it belongs to the frozen retained sensor set
#   * it has already passed corruption-operator smoke tests
#
# We include ALL sensor_2 variants:
#
#   bias    -> 6 scenarios
#   noise   -> 3 scenarios
#   drift   -> 6 scenarios
#   dropout -> 3 scenarios
#   stuck   -> 3 scenarios
#
# Total = 21 probe scenarios
#
# For bias and drift BOTH directions are retained.
# ================================================================

PROBE_SENSOR_628HC2A = "sensor_2"

if PROBE_SENSOR_628HC2A not in model_sensors_628hc2a:
    raise RuntimeError(
        f"{PROBE_SENSOR_628HC2A} is not a retained sensor."
    )

probe_manifest_628hc2a = (
    manifest_628hc2a.loc[
        manifest_628hc2a["sensor"].eq(
            PROBE_SENSOR_628HC2A
        )
    ]
    .copy()
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)

expected_probe_rows_628hc2a = 21

print("\n--- FROZEN PROBE SET ---")
print(
    "Probe sensor     :",
    PROBE_SENSOR_628HC2A,
)
print(
    "Probe scenarios  :",
    len(probe_manifest_628hc2a),
)
print(
    "Expected         :",
    expected_probe_rows_628hc2a,
)

print("\nProbe family counts:")
print(
    probe_manifest_628hc2a[
        "family"
    ].value_counts().sort_index()
)


if len(probe_manifest_628hc2a) != expected_probe_rows_628hc2a:
    raise RuntimeError(
        "Expected exactly 21 sensor_2 probe scenarios."
    )


# ================================================================
# 12. SAVE CLEAN DEV BASE FINGERPRINT
# ================================================================

base_fingerprint_before_628hc2a = (
    dev_corruption_base_628hc2a[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2a,
        ]
    ]
    .to_numpy(copy=True)
)


# ================================================================
# 13. EXPECTED DETECTOR BRANCH BY CORRUPTION FAMILY
# ================================================================

EXPECTED_BRANCH_628HC2A = {
    "bias": "level",
    "drift": "level",
    "noise": "noise",
    "dropout": "dropout",
    "stuck": "stuck",
}


# ================================================================
# 14. RUN 21 LABEL-FREE PROBE SCENARIOS
# ================================================================

clean_branch_medians_628hc2a = {
    branch: float(
        np.median(
            CLEAN_DEV_SENSOR_QUALITY_628HC2A[
                branch
            ]
        )
    )
    for branch
    in QUALITY_BRANCH_NAMES_628HC2A
}


probe_results_628hc2a = []

probe_start_628hc2a = time.perf_counter()

print(
    "\n--- RUNNING LABEL-FREE SENSOR-QUALITY PROBES ---"
)


for probe_idx, row in probe_manifest_628hc2a.iterrows():

    scenario_id = str(
        row["scenario_id"]
    )

    family = str(
        row["family"]
    )

    sensor = str(
        row["sensor"]
    )

    severity = str(
        row["severity"]
    )

    expected_branch = (
        EXPECTED_BRANCH_628HC2A[
            family
        ]
    )

    # ------------------------------------------------------------
    # Apply corruption to raw DEV copy
    # ------------------------------------------------------------

    corrupted_raw = (
        apply_scenario_628hc0(
            dev_corruption_base_628hc2a,
            row,
        )
    )

    if len(corrupted_raw) != len(
        dev_corruption_base_628hc2a
    ):
        raise RuntimeError(
            f"{scenario_id}: corrupted raw row count changed."
        )

    # ------------------------------------------------------------
    # Normalize corrupted sensor using existing frozen FIT-based
    # preprocessing bridge
    # ------------------------------------------------------------

    corrupted_normalized = (
        normalize_corrupted_sensor_628ha(
            corrupted_raw,
            sensor,
        )
    )

    # ------------------------------------------------------------
    # Rebuild the exact DEV sliding windows
    # ------------------------------------------------------------

    X_corrupted = (
        build_windows_628hc0(
            corrupted_normalized
        )
    )

    X_corrupted = np.asarray(
        X_corrupted
    )

    if X_corrupted.shape != (
        8367,
        30,
        17,
    ):
        raise RuntimeError(
            f"{scenario_id}: unexpected corrupted "
            f"window shape {X_corrupted.shape}."
        )

    if not np.isfinite(
        X_corrupted
    ).all():
        raise RuntimeError(
            f"{scenario_id}: non-finite corrupted windows."
        )

    # ------------------------------------------------------------
    # Label-free quality branches
    # ------------------------------------------------------------

    branch_scores = (
        sensor_quality_branches_628hc2a(
            X_corrupted
        )
    )

    expected_values = np.asarray(
        branch_scores[
            expected_branch
        ]
    )

    expected_threshold = (
        FIT_Q95_MARKERS_628HC2A[
            expected_branch
        ]
    )

    result_row = {
        "scenario_id": scenario_id,
        "scenario_number": int(
            row["scenario_number"]
        ),
        "family": family,
        "sensor": sensor,
        "severity": severity,
        "direction": (
            float(row["direction"])
            if pd.notna(
                row["direction"]
            )
            else np.nan
        ),
        "alpha": (
            float(row["alpha"])
            if pd.notna(
                row["alpha"]
            )
            else np.nan
        ),
        "requested_fraction": (
            float(
                row["requested_fraction"]
            )
            if pd.notna(
                row["requested_fraction"]
            )
            else np.nan
        ),
        "expected_branch": expected_branch,

        "expected_branch_median": float(
            np.median(
                expected_values
            )
        ),

        "expected_branch_q95": float(
            np.quantile(
                expected_values,
                0.95,
                method="linear",
            )
        ),

        "expected_branch_median_delta_vs_clean": float(
            np.median(
                expected_values
            )
            - clean_branch_medians_628hc2a[
                expected_branch
            ]
        ),

        "expected_branch_fraction_above_fit_q95": float(
            np.mean(
                expected_values
                > expected_threshold
            )
        ),

        "level_median": float(
            np.median(
                branch_scores[
                    "level"
                ]
            )
        ),

        "noise_median": float(
            np.median(
                branch_scores[
                    "noise"
                ]
            )
        ),

        "dropout_median": float(
            np.median(
                branch_scores[
                    "dropout"
                ]
            )
        ),

        "stuck_median": float(
            np.median(
                branch_scores[
                    "stuck"
                ]
            )
        ),

        "overall_raw_median": float(
            np.median(
                branch_scores[
                    "overall_raw"
                ]
            )
        ),
    }

    probe_results_628hc2a.append(
        result_row
    )

    print(
        f"[{probe_idx + 1:02d}/"
        f"{len(probe_manifest_628hc2a):02d}] "
        f"{scenario_id} | "
        f"{family:<7} | "
        f"{severity:<6} | "
        f"expected={expected_branch:<7} | "
        f"median="
        f"{result_row['expected_branch_median']:.4f} | "
        f"Δclean="
        f"{result_row['expected_branch_median_delta_vs_clean']:+.4f} | "
        f">FITq95="
        f"{result_row['expected_branch_fraction_above_fit_q95']:.3f}"
    )


probe_runtime_628hc2a = (
    time.perf_counter()
    - probe_start_628hc2a
)

FD002_SENSOR_QUALITY_PROBE_628HC2A = (
    pd.DataFrame(
        probe_results_628hc2a
    )
)


# ================================================================
# 15. VERIFY CLEAN SOURCE WAS NOT MODIFIED
# ================================================================

base_fingerprint_after_628hc2a = (
    dev_corruption_base_628hc2a[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2a,
        ]
    ]
    .to_numpy(copy=True)
)

clean_source_unchanged_628hc2a = bool(
    np.array_equal(
        base_fingerprint_before_628hc2a,
        base_fingerprint_after_628hc2a,
        equal_nan=True,
    )
)

if not clean_source_unchanged_628hc2a:
    raise RuntimeError(
        "The clean DEV corruption base was modified."
    )


# ================================================================
# 16. DISPLAY COMPLETE PROBE TABLE
# ================================================================

print("\n--- COMPLETE PROBE RESPONSE ---")

probe_display_columns_628hc2a = [
    "scenario_id",
    "family",
    "severity",
    "direction",
    "expected_branch",
    "expected_branch_median",
    "expected_branch_median_delta_vs_clean",
    "expected_branch_fraction_above_fit_q95",
    "level_median",
    "noise_median",
    "dropout_median",
    "stuck_median",
]

display(
    FD002_SENSOR_QUALITY_PROBE_628HC2A[
        probe_display_columns_628hc2a
    ]
)


# ================================================================
# 17. FAMILY × SEVERITY DIAGNOSTIC SUMMARY
# ================================================================

FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A = (
    FD002_SENSOR_QUALITY_PROBE_628HC2A
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        probe_scenarios=(
            "scenario_id",
            "count",
        ),

        median_expected_branch_score=(
            "expected_branch_median",
            "median",
        ),

        mean_expected_branch_score=(
            "expected_branch_median",
            "mean",
        ),

        median_delta_vs_clean=(
            "expected_branch_median_delta_vs_clean",
            "median",
        ),

        mean_fraction_above_fit_q95=(
            "expected_branch_fraction_above_fit_q95",
            "mean",
        ),

        median_level_score=(
            "level_median",
            "median",
        ),

        median_noise_score=(
            "noise_median",
            "median",
        ),

        median_dropout_score=(
            "dropout_median",
            "median",
        ),

        median_stuck_score=(
            "stuck_median",
            "median",
        ),
    )
)


severity_order_628hc2a = pd.CategoricalDtype(
    categories=[
        "low",
        "medium",
        "high",
    ],
    ordered=True,
)

family_order_628hc2a = pd.CategoricalDtype(
    categories=[
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ],
    ordered=True,
)

FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A[
    "severity"
] = (
    FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A[
        "severity"
    ].astype(
        severity_order_628hc2a
    )
)

FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A[
    "family"
] = (
    FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A[
        "family"
    ].astype(
        family_order_628hc2a
    )
)

FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A = (
    FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A
    .sort_values(
        [
            "family",
            "severity",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n--- FAMILY × SEVERITY LABEL-FREE RESPONSE ---"
)

display(
    FD002_SENSOR_QUALITY_PROBE_SUMMARY_628HC2A
)


# ================================================================
# 18. BASIC NUMERICAL / METHODOLOGY AUDIT
# ================================================================

all_probe_numeric_finite_628hc2a = bool(
    np.isfinite(
        FD002_SENSOR_QUALITY_PROBE_628HC2A[
            [
                "expected_branch_median",
                "expected_branch_q95",
                "expected_branch_median_delta_vs_clean",
                "expected_branch_fraction_above_fit_q95",
                "level_median",
                "noise_median",
                "dropout_median",
                "stuck_median",
                "overall_raw_median",
            ]
        ].to_numpy()
    ).all()
)

all_probe_scores_in_range_628hc2a = bool(
    (
        FD002_SENSOR_QUALITY_PROBE_628HC2A[
            [
                "expected_branch_median",
                "expected_branch_q95",
                "expected_branch_fraction_above_fit_q95",
                "level_median",
                "noise_median",
                "dropout_median",
                "stuck_median",
                "overall_raw_median",
            ]
        ]
        .to_numpy()
        >= 0.0
    ).all()
    and
    (
        FD002_SENSOR_QUALITY_PROBE_628HC2A[
            [
                "expected_branch_median",
                "expected_branch_q95",
                "expected_branch_fraction_above_fit_q95",
                "level_median",
                "noise_median",
                "dropout_median",
                "stuck_median",
                "overall_raw_median",
            ]
        ]
        .to_numpy()
        <= 1.0
    ).all()
)


if not all_probe_numeric_finite_628hc2a:
    raise RuntimeError(
        "Probe table contains non-finite quality values."
    )

if not all_probe_scores_in_range_628hc2a:
    raise RuntimeError(
        "Probe quality values fall outside [0,1]."
    )


# ================================================================
# 19. FREEZE PRE-FLIGHT CONTRACT
# ================================================================

FD002_SENSOR_QUALITY_REFERENCE_628HC2A = {
    "reference_population": "FD002_FIT_windows_only",
    "fit_windows": int(
        X_fit_628hc2a.shape[0]
    ),
    "sensors": list(
        model_sensors_628hc2a
    ),
    "statistics": [
        "absolute_window_mean",
        "population_window_std_ddof_0",
        "mean_absolute_first_difference",
        "normalized_zero_fraction",
        "flat_step_fraction",
    ],
    "branch_level": (
        "high_tail_absolute_window_mean"
    ),
    "branch_noise": (
        "max(high_tail_STD, "
        "high_tail_temporal_roughness)"
    ),
    "branch_dropout": (
        "high_tail_normalized_zero_fraction"
    ),
    "branch_stuck": (
        "max(low_tail_STD, "
        "low_tail_temporal_roughness, "
        "high_tail_flat_step_fraction)"
    ),
    "empirical_transform": (
        "sensor_specific_FIT_midrank_percentile"
    ),
    "zero_tolerance": ZERO_TOL_628HC2A,
    "flat_tolerance": FLAT_TOL_628HC2A,
    "fit_q95_markers": dict(
        FIT_Q95_MARKERS_628HC2A
    ),
    "abstention_threshold_selected": False,
    "dev_rul_labels_used": False,
    "official_test_labels_read": False,
    "test_scored": False,
    "model_retrained": False,
    "files_written": False,
}


FD002_SENSOR_QUALITY_PREFLIGHT_CONTRACT_628HC2A = {
    "probe_sensor": PROBE_SENSOR_628HC2A,
    "probe_scenarios": int(
        len(
            FD002_SENSOR_QUALITY_PROBE_628HC2A
        )
    ),
    "expected_probe_scenarios": 21,
    "fit_reference_only": True,
    "clean_dev_source_unchanged": (
        clean_source_unchanged_628hc2a
    ),
    "all_probe_values_finite": (
        all_probe_numeric_finite_628hc2a
    ),
    "all_probe_scores_within_0_1": (
        all_probe_scores_in_range_628hc2a
    ),
    "dev_rul_labels_used": False,
    "official_test_labels_read": False,
    "test_scored": False,
    "abstention_threshold_selected": False,
    "model_retrained": False,
    "files_written": False,
}


# ================================================================
# 20. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 68)
print("STEP 6.28H-C2-A FINAL SUMMARY")
print("=" * 68)

print(
    "FIT reference windows          :",
    X_fit_628hc2a.shape[0],
)

print(
    "Clean DEV diagnostic windows   :",
    X_dev_clean_628hc2a.shape[0],
)

print(
    "Retained sensors               :",
    len(model_sensors_628hc2a),
)

print(
    "Sensor-quality branches        :",
    4,
)

print(
    "Probe sensor                   :",
    PROBE_SENSOR_628HC2A,
)

print(
    "Probe scenarios                :",
    len(
        FD002_SENSOR_QUALITY_PROBE_628HC2A
    ),
)

print(
    "FIT-only statistical reference :",
    True,
)

print(
    "Clean DEV source unchanged     :",
    clean_source_unchanged_628hc2a,
)

print(
    "All probe values finite        :",
    all_probe_numeric_finite_628hc2a,
)

print(
    "All probe scores within [0,1]  :",
    all_probe_scores_in_range_628hc2a,
)

print(
    "Pre-flight runtime (sec)       :",
    probe_runtime_628hc2a
    + clean_quality_runtime_628hc2a
    + fit_reference_runtime_628hc2a,
)

print(
    "DEV RUL labels used            :",
    False,
)

print(
    "Official TEST labels read      :",
    False,
)

print(
    "TEST scored                    :",
    False,
)

print(
    "Abstention threshold selected  :",
    False,
)

print(
    "Model retrained                :",
    False,
)

print(
    "Files written                  :",
    False,
)

print(
    "\n=== END STEP 6.28H-C2-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C2-B — FIT-CALIBRATED SENSOR-QUALITY RISK
# ================================================================
#
# PURPOSE
# -------
# Correct the max-over-sensors / max-over-branches calibration issue
# identified in Step 6.28H-C2-A.
#
# Pipeline:
#
# sensor statistic
#      ↓
# sensor-specific FIT percentile
#      ↓
# branch raw maximum across sensors
#      ↓
# FIT calibration of branch maximum
#      ↓
# maximum across calibrated branches
#      ↓
# FIT calibration of that maximum
#      ↓
# FINAL LABEL-FREE SENSOR-QUALITY RISK
#
# IMPORTANT
# ---------
# * FIT is the ONLY calibration/reference population.
# * DEV labels are NOT used.
# * Official TEST labels are NOT read.
# * No abstention threshold is selected.
# * No model retraining.
# * No files are written.
# ================================================================

import time
import numpy as np
import pandas as pd


print("=" * 72)
print("STEP 6.28H-C2-B — FIT-CALIBRATED SENSOR-QUALITY RISK")
print("=" * 72)


# ================================================================
# 1. REQUIRED OBJECTS
# ================================================================

required_objects_628hc2b = [
    "FIT_SENSOR_QUALITY_628HC2A",
    "CLEAN_DEV_SENSOR_QUALITY_628HC2A",
    "FD002_SENSOR_QUALITY_REFERENCE_628HC2A",
    "FD002_SENSOR_QUALITY_PREFLIGHT_CONTRACT_628HC2A",
    "sensor_quality_branches_628hc2a",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "X_FIT_FD002_628D",
    "X_DEV_FD002_628D",
    "FD002_MODEL_SENSORS_628GB",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc2b = []

for name in required_objects_628hc2b:
    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:
        missing_628hc2b.append(name)


if missing_628hc2b:
    raise RuntimeError(
        "Missing required Step 6.28H-C2-B objects:\n"
        + "\n".join(missing_628hc2b)
    )


# ================================================================
# 2. FROZEN INPUT CONTRACT
# ================================================================

X_fit_628hc2b = np.asarray(
    X_FIT_FD002_628D
)

X_dev_clean_628hc2b = np.asarray(
    X_DEV_FD002_628D
)

manifest_628hc2b = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
)

dev_base_628hc2b = (
    FD002_DEV_CORRUPTION_BASE_628GC
)

model_sensors_628hc2b = list(
    FD002_MODEL_SENSORS_628GB
)

BRANCHES_628HC2B = [
    "level",
    "noise",
    "dropout",
    "stuck",
]


print("\n--- POPULATION CONTRACT ---")

print(
    "FIT windows       :",
    X_fit_628hc2b.shape[0],
)

print(
    "Clean DEV windows :",
    X_dev_clean_628hc2b.shape[0],
)

print(
    "Retained sensors  :",
    len(model_sensors_628hc2b),
)

print(
    "Frozen scenarios  :",
    len(manifest_628hc2b),
)

print(
    "Branches          :",
    BRANCHES_628HC2B,
)


if X_fit_628hc2b.shape != (37852, 30, 17):
    raise RuntimeError(
        f"Unexpected FIT shape: {X_fit_628hc2b.shape}"
    )

if X_dev_clean_628hc2b.shape != (8367, 30, 17):
    raise RuntimeError(
        f"Unexpected DEV shape: {X_dev_clean_628hc2b.shape}"
    )

if len(model_sensors_628hc2b) != 15:
    raise RuntimeError(
        "Expected exactly 15 retained sensors."
    )

if len(manifest_628hc2b) != 315:
    raise RuntimeError(
        "Expected exactly 315 corruption scenarios."
    )


# ================================================================
# 3. ONE-DIMENSIONAL EMPIRICAL MID-RANK CDF
# ================================================================
#
# The input branch score is already:
#
#      maximum across sensor-specific anomaly scores
#
# We now calibrate that maximum against its empirical distribution
# on CLEAN FIT windows.
#
# This automatically accounts for the max-over-15-sensors effect.
# ================================================================

def empirical_midrank_1d_628hc2b(
    values,
    sorted_reference,
):
    """
    Empirical mid-rank CDF for a 1-D score.

    Larger values -> larger calibrated risk.

    Reference MUST be supplied from FIT only.
    """

    x = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    ref = np.asarray(
        sorted_reference,
        dtype=np.float64,
    ).reshape(-1)

    if len(ref) == 0:
        raise ValueError(
            "Reference population is empty."
        )

    if not np.isfinite(x).all():
        raise ValueError(
            "Input values contain non-finite values."
        )

    if not np.isfinite(ref).all():
        raise ValueError(
            "Reference values contain non-finite values."
        )

    left = np.searchsorted(
        ref,
        x,
        side="left",
    )

    right = np.searchsorted(
        ref,
        x,
        side="right",
    )

    result = (
        left + right
    ) / (
        2.0 * len(ref)
    )

    return np.clip(
        result,
        0.0,
        1.0,
    )


# ================================================================
# 4. BUILD FIT BRANCH-MAX REFERENCES
# ================================================================

FIT_BRANCH_MAX_REFERENCE_628HC2B = {}

for branch in BRANCHES_628HC2B:

    values = np.asarray(
        FIT_SENSOR_QUALITY_628HC2A[
            branch
        ],
        dtype=np.float64,
    )

    if values.shape != (37852,):
        raise RuntimeError(
            f"Unexpected FIT {branch} shape: "
            f"{values.shape}"
        )

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"FIT {branch} contains non-finite values."
        )

    FIT_BRANCH_MAX_REFERENCE_628HC2B[
        branch
    ] = np.sort(
        values
    )


print("\n--- FIT BRANCH-MAX REFERENCES ---")

for branch in BRANCHES_628HC2B:

    ref = (
        FIT_BRANCH_MAX_REFERENCE_628HC2B[
            branch
        ]
    )

    print(
        f"{branch:<10} "
        f"n={len(ref):5d} "
        f"min={np.min(ref):.6f} "
        f"median={np.median(ref):.6f} "
        f"q95={np.quantile(ref, 0.95):.6f} "
        f"max={np.max(ref):.6f}"
    )


# ================================================================
# 5. FIRST-STAGE BRANCH CALIBRATION
# ================================================================

def calibrate_branch_scores_628hc2b(
    raw_branch_dict
):
    """
    FIT-calibrate each raw branch maximum independently.

    Output values are in [0,1].
    """

    output = {}

    expected_n = None

    for branch in BRANCHES_628HC2B:

        values = np.asarray(
            raw_branch_dict[
                branch
            ],
            dtype=np.float64,
        ).reshape(-1)

        if expected_n is None:
            expected_n = len(values)

        if len(values) != expected_n:
            raise RuntimeError(
                "Branch score lengths do not agree."
            )

        output[branch] = (
            empirical_midrank_1d_628hc2b(
                values,
                FIT_BRANCH_MAX_REFERENCE_628HC2B[
                    branch
                ],
            )
        )

    return output


# ================================================================
# 6. FIT FIRST-STAGE CALIBRATED BRANCH SCORES
# ================================================================

FIT_CALIBRATED_BRANCHES_628HC2B = (
    calibrate_branch_scores_628hc2b(
        FIT_SENSOR_QUALITY_628HC2A
    )
)


for branch in BRANCHES_628HC2B:

    values = (
        FIT_CALIBRATED_BRANCHES_628HC2B[
            branch
        ]
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Calibrated FIT {branch} is not finite."
        )

    if (
        np.min(values) < 0.0
        or np.max(values) > 1.0
    ):
        raise RuntimeError(
            f"Calibrated FIT {branch} outside [0,1]."
        )


# ================================================================
# 7. CREATE FIT MULTI-BRANCH MAXIMUM
# ================================================================
#
# We intentionally do not choose hand-made branch weights here.
#
# Any branch can flag a potentially unreliable input.
#
# After taking the maximum, we calibrate AGAIN against FIT.
# This corrects for searching across four branches.
# ================================================================

FIT_PRECALIBRATED_GLOBAL_628HC2B = (
    np.maximum.reduce(
        [
            FIT_CALIBRATED_BRANCHES_628HC2B[
                branch
            ]
            for branch
            in BRANCHES_628HC2B
        ]
    )
)

FIT_GLOBAL_REFERENCE_628HC2B = (
    np.sort(
        FIT_PRECALIBRATED_GLOBAL_628HC2B
    )
)


# ================================================================
# 8. FINAL SENSOR-QUALITY CALIBRATION FUNCTION
# ================================================================

def final_sensor_quality_from_raw_628hc2b(
    raw_branch_dict
):
    """
    Produce:
        calibrated branch risks
        pre-global risk
        final FIT-calibrated sensor-quality risk
    """

    calibrated = (
        calibrate_branch_scores_628hc2b(
            raw_branch_dict
        )
    )

    pre_global = (
        np.maximum.reduce(
            [
                calibrated[
                    branch
                ]
                for branch
                in BRANCHES_628HC2B
            ]
        )
    )

    final_risk = (
        empirical_midrank_1d_628hc2b(
            pre_global,
            FIT_GLOBAL_REFERENCE_628HC2B,
        )
    )

    result = {
        "level": calibrated["level"],
        "noise": calibrated["noise"],
        "dropout": calibrated["dropout"],
        "stuck": calibrated["stuck"],
        "pre_global": pre_global,
        "sensor_quality_risk": final_risk,
    }

    for name, values in result.items():

        values = np.asarray(values)

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{name} contains non-finite values."
            )

        if (
            np.min(values) < 0.0
            or np.max(values) > 1.0
        ):
            raise RuntimeError(
                f"{name} falls outside [0,1]."
            )

    return result


# ================================================================
# 9. APPLY FINAL CALIBRATION TO FIT
# ================================================================

FIT_SENSOR_QUALITY_CALIBRATED_628HC2B = (
    final_sensor_quality_from_raw_628hc2b(
        FIT_SENSOR_QUALITY_628HC2A
    )
)


# ================================================================
# 10. APPLY FINAL CALIBRATION TO CLEAN DEV
# ================================================================

CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B = (
    final_sensor_quality_from_raw_628hc2b(
        CLEAN_DEV_SENSOR_QUALITY_628HC2A
    )
)


# ================================================================
# 11. FINAL FIT Q95 DIAGNOSTIC MARKER
# ================================================================
#
# Again:
#
# this is NOT an abstention threshold.
#
# It is only a reference diagnostic marker.
# ================================================================

FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B = float(
    np.quantile(
        FIT_SENSOR_QUALITY_CALIBRATED_628HC2B[
            "sensor_quality_risk"
        ],
        0.95,
        method="linear",
    )
)


# ================================================================
# 12. CLEAN FIT / DEV CALIBRATION SUMMARY
# ================================================================

def risk_summary_628hc2b(
    population_name,
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "population": population_name,
        "n": len(values),
        "minimum": float(
            np.min(values)
        ),
        "q25": float(
            np.quantile(
                values,
                0.25,
                method="linear",
            )
        ),
        "median": float(
            np.median(values)
        ),
        "mean": float(
            np.mean(values)
        ),
        "q75": float(
            np.quantile(
                values,
                0.75,
                method="linear",
            )
        ),
        "q95": float(
            np.quantile(
                values,
                0.95,
                method="linear",
            )
        ),
        "maximum": float(
            np.max(values)
        ),
        "fraction_above_fit_q95": float(
            np.mean(
                values
                > FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
            )
        ),
    }


FD002_SENSOR_QUALITY_CLEAN_CALIBRATION_628HC2B = (
    pd.DataFrame(
        [
            risk_summary_628hc2b(
                "FIT",
                FIT_SENSOR_QUALITY_CALIBRATED_628HC2B[
                    "sensor_quality_risk"
                ],
            ),
            risk_summary_628hc2b(
                "DEV",
                CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
                    "sensor_quality_risk"
                ],
            ),
        ]
    )
)


print(
    "\n--- CLEAN FIT / DEV FINAL SENSOR-QUALITY RISK ---"
)

display(
    FD002_SENSOR_QUALITY_CLEAN_CALIBRATION_628HC2B
)

print(
    "\nFIT final-risk q95 diagnostic marker:",
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B,
)


# ================================================================
# 13. PROBE SET — SENSOR_2 ONLY
# ================================================================

PROBE_SENSOR_628HC2B = "sensor_2"

probe_manifest_628hc2b = (
    manifest_628hc2b.loc[
        manifest_628hc2b[
            "sensor"
        ].eq(
            PROBE_SENSOR_628HC2B
        )
    ]
    .copy()
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)

if len(probe_manifest_628hc2b) != 21:
    raise RuntimeError(
        "Expected 21 sensor_2 probe scenarios."
    )


EXPECTED_BRANCH_628HC2B = {
    "bias": "level",
    "noise": "noise",
    "drift": "level",
    "dropout": "dropout",
    "stuck": "stuck",
}


# ================================================================
# 14. PRESERVE CLEAN SOURCE FINGERPRINT
# ================================================================

clean_source_before_628hc2b = (
    dev_base_628hc2b[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2b,
        ]
    ]
    .to_numpy(copy=True)
)


# ================================================================
# 15. RUN CALIBRATED PROBES
# ================================================================

probe_rows_628hc2b = []

probe_start_628hc2b = time.perf_counter()


print(
    "\n--- RUNNING 21 FIT-CALIBRATED QUALITY PROBES ---"
)


for probe_index, row in probe_manifest_628hc2b.iterrows():

    scenario_id = str(
        row["scenario_id"]
    )

    family = str(
        row["family"]
    )

    severity = str(
        row["severity"]
    )

    sensor = str(
        row["sensor"]
    )

    expected_branch = (
        EXPECTED_BRANCH_628HC2B[
            family
        ]
    )

    # ------------------------------------------------------------
    # Apply frozen corruption
    # ------------------------------------------------------------

    corrupted_raw = (
        apply_scenario_628hc0(
            dev_base_628hc2b,
            row,
        )
    )

    # ------------------------------------------------------------
    # Frozen FIT-based normalization
    # ------------------------------------------------------------

    corrupted_normalized = (
        normalize_corrupted_sensor_628ha(
            corrupted_raw,
            sensor,
        )
    )

    # ------------------------------------------------------------
    # Exact existing DEV window bridge
    # ------------------------------------------------------------

    X_corrupted = np.asarray(
        build_windows_628hc0(
            corrupted_normalized
        )
    )

    if X_corrupted.shape != (
        8367,
        30,
        17,
    ):
        raise RuntimeError(
            f"{scenario_id}: bad window shape "
            f"{X_corrupted.shape}"
        )

    if not np.isfinite(
        X_corrupted
    ).all():
        raise RuntimeError(
            f"{scenario_id}: corrupted windows "
            "contain non-finite values."
        )

    # ------------------------------------------------------------
    # Raw label-free branch statistics
    # ------------------------------------------------------------

    raw_scores = (
        sensor_quality_branches_628hc2a(
            X_corrupted
        )
    )

    # ------------------------------------------------------------
    # FIT calibration
    # ------------------------------------------------------------

    calibrated_scores = (
        final_sensor_quality_from_raw_628hc2b(
            raw_scores
        )
    )

    expected_values = np.asarray(
        calibrated_scores[
            expected_branch
        ]
    )

    final_quality = np.asarray(
        calibrated_scores[
            "sensor_quality_risk"
        ]
    )

    result = {
        "scenario_id": scenario_id,

        "scenario_number": int(
            row["scenario_number"]
        ),

        "family": family,

        "sensor": sensor,

        "severity": severity,

        "direction": (
            float(row["direction"])
            if pd.notna(
                row["direction"]
            )
            else np.nan
        ),

        "alpha": (
            float(row["alpha"])
            if pd.notna(
                row["alpha"]
            )
            else np.nan
        ),

        "requested_fraction": (
            float(
                row["requested_fraction"]
            )
            if pd.notna(
                row["requested_fraction"]
            )
            else np.nan
        ),

        "expected_branch": expected_branch,

        "expected_branch_median": float(
            np.median(
                expected_values
            )
        ),

        "expected_branch_mean": float(
            np.mean(
                expected_values
            )
        ),

        "final_quality_median": float(
            np.median(
                final_quality
            )
        ),

        "final_quality_mean": float(
            np.mean(
                final_quality
            )
        ),

        "final_quality_q95": float(
            np.quantile(
                final_quality,
                0.95,
                method="linear",
            )
        ),

        "fraction_above_fit_q95": float(
            np.mean(
                final_quality
                > FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
            )
        ),

        "level_median": float(
            np.median(
                calibrated_scores[
                    "level"
                ]
            )
        ),

        "noise_median": float(
            np.median(
                calibrated_scores[
                    "noise"
                ]
            )
        ),

        "dropout_median": float(
            np.median(
                calibrated_scores[
                    "dropout"
                ]
            )
        ),

        "stuck_median": float(
            np.median(
                calibrated_scores[
                    "stuck"
                ]
            )
        ),
    }

    probe_rows_628hc2b.append(
        result
    )

    print(
        f"[{probe_index + 1:02d}/21] "
        f"{scenario_id} | "
        f"{family:<7} | "
        f"{severity:<6} | "
        f"branch={expected_branch:<7} | "
        f"branch_med="
        f"{result['expected_branch_median']:.4f} | "
        f"final_med="
        f"{result['final_quality_median']:.4f} | "
        f">FITq95="
        f"{result['fraction_above_fit_q95']:.3f}"
    )


probe_runtime_628hc2b = (
    time.perf_counter()
    - probe_start_628hc2b
)


FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B = (
    pd.DataFrame(
        probe_rows_628hc2b
    )
)


# ================================================================
# 16. VERIFY CLEAN SOURCE UNCHANGED
# ================================================================

clean_source_after_628hc2b = (
    dev_base_628hc2b[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2b,
        ]
    ]
    .to_numpy(copy=True)
)


clean_source_unchanged_628hc2b = bool(
    np.array_equal(
        clean_source_before_628hc2b,
        clean_source_after_628hc2b,
        equal_nan=True,
    )
)


if not clean_source_unchanged_628hc2b:
    raise RuntimeError(
        "Clean DEV corruption base was modified."
    )


# ================================================================
# 17. FAMILY × SEVERITY CALIBRATED RESPONSE
# ================================================================

FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B = (
    FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        median_expected_branch=(
            "expected_branch_median",
            "median",
        ),

        mean_expected_branch=(
            "expected_branch_mean",
            "mean",
        ),

        median_final_quality=(
            "final_quality_median",
            "median",
        ),

        mean_final_quality=(
            "final_quality_mean",
            "mean",
        ),

        median_final_q95=(
            "final_quality_q95",
            "median",
        ),

        mean_fraction_above_fit_q95=(
            "fraction_above_fit_q95",
            "mean",
        ),
    )
)


severity_type_628hc2b = (
    pd.CategoricalDtype(
        categories=[
            "low",
            "medium",
            "high",
        ],
        ordered=True,
    )
)

family_type_628hc2b = (
    pd.CategoricalDtype(
        categories=[
            "bias",
            "noise",
            "drift",
            "dropout",
            "stuck",
        ],
        ordered=True,
    )
)


FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B[
    "severity"
] = (
    FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B[
        "severity"
    ]
    .astype(
        severity_type_628hc2b
    )
)

FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B[
    "family"
] = (
    FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B[
        "family"
    ]
    .astype(
        family_type_628hc2b
    )
)


FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B = (
    FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B
    .sort_values(
        [
            "family",
            "severity",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n--- FAMILY × SEVERITY FIT-CALIBRATED RESPONSE ---"
)

display(
    FD002_SENSOR_QUALITY_CALIBRATED_SUMMARY_628HC2B
)


# ================================================================
# 18. COMPLETE PROBE TABLE
# ================================================================

print(
    "\n--- COMPLETE FIT-CALIBRATED PROBE TABLE ---"
)

display(
    FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B[
        [
            "scenario_id",
            "family",
            "severity",
            "direction",
            "expected_branch",
            "expected_branch_median",
            "final_quality_median",
            "final_quality_mean",
            "fraction_above_fit_q95",
            "level_median",
            "noise_median",
            "dropout_median",
            "stuck_median",
        ]
    ]
)


# ================================================================
# 19. NUMERICAL INTEGRITY
# ================================================================

numeric_columns_628hc2b = [
    "expected_branch_median",
    "expected_branch_mean",
    "final_quality_median",
    "final_quality_mean",
    "final_quality_q95",
    "fraction_above_fit_q95",
    "level_median",
    "noise_median",
    "dropout_median",
    "stuck_median",
]


probe_numeric_628hc2b = (
    FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B[
        numeric_columns_628hc2b
    ]
    .to_numpy(
        dtype=float
    )
)


all_finite_628hc2b = bool(
    np.isfinite(
        probe_numeric_628hc2b
    ).all()
)


all_in_range_628hc2b = bool(
    (
        probe_numeric_628hc2b
        >= 0.0
    ).all()
    and
    (
        probe_numeric_628hc2b
        <= 1.0
    ).all()
)


if not all_finite_628hc2b:
    raise RuntimeError(
        "Calibrated probe output contains "
        "non-finite values."
    )


if not all_in_range_628hc2b:
    raise RuntimeError(
        "Calibrated probe output falls "
        "outside [0,1]."
    )


# ================================================================
# 20. FREEZE CALIBRATION CONTRACT
# ================================================================

FD002_SENSOR_QUALITY_CALIBRATION_CONTRACT_628HC2B = {
    "dataset": "FD002",

    "purpose": (
        "label_free_sensor_quality_risk"
    ),

    "sensor_reference": (
        "FD002_FIT_only"
    ),

    "sensor_specific_transform": (
        "empirical_midrank_percentile"
    ),

    "branch_aggregation": (
        "maximum_across_15_retained_sensors"
    ),

    "branch_max_calibration": (
        "FIT_empirical_midrank_CDF"
    ),

    "branches": list(
        BRANCHES_628HC2B
    ),

    "global_precalibration": (
        "maximum_across_calibrated_branches"
    ),

    "global_calibration": (
        "FIT_empirical_midrank_CDF"
    ),

    "fit_reference_windows": int(
        X_fit_628hc2b.shape[0]
    ),

    "clean_dev_windows": int(
        X_dev_clean_628hc2b.shape[0]
    ),

    "probe_scenarios": int(
        len(
            FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B
        )
    ),

    "fit_q95_diagnostic_marker": (
        FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
    ),

    "clean_dev_source_unchanged": (
        clean_source_unchanged_628hc2b
    ),

    "all_probe_values_finite": (
        all_finite_628hc2b
    ),

    "all_probe_values_within_0_1": (
        all_in_range_628hc2b
    ),

    "dev_rul_labels_used": False,

    "official_test_labels_read": False,

    "test_scored": False,

    "abstention_threshold_selected": False,

    "model_retrained": False,

    "files_written": False,
}


# ================================================================
# 21. FINAL SUMMARY
# ================================================================

fit_final_628hc2b = np.asarray(
    FIT_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "sensor_quality_risk"
    ]
)

dev_final_628hc2b = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "sensor_quality_risk"
    ]
)


print("\n" + "=" * 72)
print("STEP 6.28H-C2-B FINAL SUMMARY")
print("=" * 72)

print(
    "FIT reference windows             :",
    X_fit_628hc2b.shape[0],
)

print(
    "Clean DEV windows                 :",
    X_dev_clean_628hc2b.shape[0],
)

print(
    "Retained sensors                  :",
    len(model_sensors_628hc2b),
)

print(
    "Calibrated sensor-quality branches:",
    len(BRANCHES_628HC2B),
)

print(
    "FIT final-risk median             :",
    float(
        np.median(
            fit_final_628hc2b
        )
    ),
)

print(
    "Clean DEV final-risk median       :",
    float(
        np.median(
            dev_final_628hc2b
        )
    ),
)

print(
    "FIT q95 diagnostic marker         :",
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B,
)

print(
    "Clean DEV fraction > FIT q95      :",
    float(
        np.mean(
            dev_final_628hc2b
            > FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
        )
    ),
)

print(
    "Probe scenarios                   :",
    len(
        FD002_SENSOR_QUALITY_PROBE_CALIBRATED_628HC2B
    ),
)

print(
    "Clean DEV source unchanged        :",
    clean_source_unchanged_628hc2b,
)

print(
    "All probe values finite           :",
    all_finite_628hc2b,
)

print(
    "All probe values within [0,1]     :",
    all_in_range_628hc2b,
)

print(
    "Probe runtime (sec)               :",
    probe_runtime_628hc2b,
)

print(
    "DEV RUL labels used               :",
    False,
)

print(
    "Official TEST labels read         :",
    False,
)

print(
    "TEST scored                       :",
    False,
)

print(
    "Abstention threshold selected     :",
    False,
)

print(
    "Model retrained                   :",
    False,
)

print(
    "Files written                     :",
    False,
)

print(
    "\n=== END STEP 6.28H-C2-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C2-C — FULL FD002 SENSOR-QUALITY RESPONSE
# ================================================================
#
# PURPOSE
# -------
# Apply the frozen, FIT-calibrated label-free sensor-quality
# detector from Step 6.28H-C2-B to ALL 315 frozen FD002
# corruption scenarios.
#
# Produces:
#
#   1. one sensor-quality risk vector for every scenario
#   2. full risk matrix: (315, 8367)
#   3. scenario-level summaries
#   4. family × severity summaries across ALL 15 sensors
#
# IMPORTANT
# ---------
# * Calibration remains FIT-only.
# * No DEV RUL labels are used.
# * No official TEST labels are read.
# * No abstention threshold is selected.
# * No model inference is required here.
# * No model retraining.
# * No files are written.
# ================================================================

import time
import gc
import numpy as np
import pandas as pd


print("=" * 74)
print("STEP 6.28H-C2-C — FULL FD002 SENSOR-QUALITY RESPONSE")
print("=" * 74)


# ================================================================
# 1. REQUIRED OBJECTS
# ================================================================

required_objects_628hc2c = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_MODEL_SENSORS_628GB",
    "X_DEV_FD002_628D",
    "FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B",
    "FIT_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FD002_SENSOR_QUALITY_CALIBRATION_CONTRACT_628HC2B",
    "sensor_quality_branches_628hc2a",
    "final_sensor_quality_from_raw_628hc2b",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc2c = []

for name in required_objects_628hc2c:
    exists = name in globals()
    print(f"{name:<55}: {exists}")

    if not exists:
        missing_628hc2c.append(name)

if missing_628hc2c:
    raise RuntimeError(
        "Missing Step 6.28H-C2-C objects:\n"
        + "\n".join(missing_628hc2c)
    )


# ================================================================
# 2. FROZEN INPUTS
# ================================================================

manifest_628hc2c = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values("scenario_number")
    .reset_index(drop=True)
)

dev_base_628hc2c = (
    FD002_DEV_CORRUPTION_BASE_628GC
)

model_sensors_628hc2c = list(
    FD002_MODEL_SENSORS_628GB
)

clean_dev_X_628hc2c = np.asarray(
    X_DEV_FD002_628D
)

clean_quality_628hc2c = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "sensor_quality_risk"
    ],
    dtype=np.float64,
)


N_SCENARIOS_628HC2C = 315
N_WINDOWS_628HC2C = 8367


print("\n--- FROZEN EXPERIMENT CONTRACT ---")

print(
    "Frozen scenarios        :",
    len(manifest_628hc2c),
)

print(
    "DEV windows             :",
    clean_dev_X_628hc2c.shape[0],
)

print(
    "Window shape            :",
    clean_dev_X_628hc2c.shape,
)

print(
    "Retained sensors        :",
    len(model_sensors_628hc2c),
)

print(
    "FIT q95 marker          :",
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B,
)

print(
    "Clean DEV quality count :",
    len(clean_quality_628hc2c),
)


if len(manifest_628hc2c) != N_SCENARIOS_628HC2C:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )

if clean_dev_X_628hc2c.shape != (8367, 30, 17):
    raise RuntimeError(
        f"Unexpected clean DEV X shape: "
        f"{clean_dev_X_628hc2c.shape}"
    )

if len(model_sensors_628hc2c) != 15:
    raise RuntimeError(
        "Expected exactly 15 retained sensors."
    )

if clean_quality_628hc2c.shape != (8367,):
    raise RuntimeError(
        "Unexpected clean sensor-quality risk shape."
    )

if not np.isfinite(clean_quality_628hc2c).all():
    raise RuntimeError(
        "Clean DEV quality risk is not finite."
    )

if (
    clean_quality_628hc2c.min() < 0.0
    or clean_quality_628hc2c.max() > 1.0
):
    raise RuntimeError(
        "Clean DEV quality risk outside [0,1]."
    )


# ================================================================
# 3. MANIFEST ORDER CONTRACT
# ================================================================

expected_numbers_628hc2c = np.arange(
    1,
    N_SCENARIOS_628HC2C + 1,
)

actual_numbers_628hc2c = (
    manifest_628hc2c[
        "scenario_number"
    ]
    .to_numpy(dtype=int)
)


scenario_number_exact_628hc2c = bool(
    np.array_equal(
        expected_numbers_628hc2c,
        actual_numbers_628hc2c,
    )
)


expected_ids_628hc2c = np.asarray(
    [
        f"FD002_SCENARIO_{i:03d}"
        for i in expected_numbers_628hc2c
    ],
    dtype=object,
)

actual_ids_628hc2c = (
    manifest_628hc2c[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)


scenario_id_exact_628hc2c = bool(
    np.array_equal(
        expected_ids_628hc2c,
        actual_ids_628hc2c,
    )
)


print("\n--- MANIFEST ORDER CONTRACT ---")

print(
    "Scenario numbers 001–315 exact :",
    scenario_number_exact_628hc2c,
)

print(
    "Scenario IDs exact             :",
    scenario_id_exact_628hc2c,
)


if not scenario_number_exact_628hc2c:
    raise RuntimeError(
        "Scenario-number ordering is not exact."
    )

if not scenario_id_exact_628hc2c:
    raise RuntimeError(
        "Scenario-ID ordering is not exact."
    )


# ================================================================
# 4. EXPECTED FAMILY BRANCH
# ================================================================

EXPECTED_BRANCH_628HC2C = {
    "bias": "level",
    "noise": "noise",
    "drift": "level",
    "dropout": "dropout",
    "stuck": "stuck",
}


# ================================================================
# 5. PRESERVE CLEAN SOURCE
# ================================================================

clean_source_before_628hc2c = (
    dev_base_628hc2c[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2c,
        ]
    ]
    .to_numpy(copy=True)
)


# ================================================================
# 6. PREALLOCATE FULL SENSOR-QUALITY MATRIX
# ================================================================

FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C = (
    np.empty(
        (
            N_SCENARIOS_628HC2C,
            N_WINDOWS_628HC2C,
        ),
        dtype=np.float64,
    )
)


scenario_rows_628hc2c = []


# ================================================================
# 7. RUN ALL 315 SCENARIOS
# ================================================================

print(
    "\n--- RUNNING 315 FIT-CALIBRATED SENSOR-QUALITY SCENARIOS ---"
)

start_628hc2c = time.perf_counter()


for i, row in manifest_628hc2c.iterrows():

    scenario_start = time.perf_counter()

    scenario_id = str(
        row["scenario_id"]
    )

    family = str(
        row["family"]
    )

    sensor = str(
        row["sensor"]
    )

    severity = str(
        row["severity"]
    )

    expected_branch = (
        EXPECTED_BRANCH_628HC2C[
            family
        ]
    )

    # ------------------------------------------------------------
    # A. Apply frozen raw-data corruption
    # ------------------------------------------------------------

    corrupted_raw_628hc2c = (
        apply_scenario_628hc0(
            dev_base_628hc2c,
            row,
        )
    )


    # ------------------------------------------------------------
    # B. Apply frozen FIT-derived normalization
    # ------------------------------------------------------------

    corrupted_normalized_628hc2c = (
        normalize_corrupted_sensor_628ha(
            corrupted_raw_628hc2c,
            sensor,
        )
    )


    # ------------------------------------------------------------
    # C. Build exact 30-cycle model windows
    # ------------------------------------------------------------

    X_corrupted_628hc2c = np.asarray(
        build_windows_628hc0(
            corrupted_normalized_628hc2c
        )
    )


    if X_corrupted_628hc2c.shape != (
        N_WINDOWS_628HC2C,
        30,
        17,
    ):
        raise RuntimeError(
            f"{scenario_id}: unexpected window shape "
            f"{X_corrupted_628hc2c.shape}"
        )

    if not np.isfinite(
        X_corrupted_628hc2c
    ).all():
        raise RuntimeError(
            f"{scenario_id}: corrupted windows "
            "contain non-finite values."
        )


    # ------------------------------------------------------------
    # D. Compute frozen raw sensor-quality branches
    # ------------------------------------------------------------

    raw_quality_628hc2c = (
        sensor_quality_branches_628hc2a(
            X_corrupted_628hc2c
        )
    )


    # ------------------------------------------------------------
    # E. Apply frozen FIT calibration
    # ------------------------------------------------------------

    calibrated_quality_628hc2c = (
        final_sensor_quality_from_raw_628hc2b(
            raw_quality_628hc2c
        )
    )


    final_risk_628hc2c = np.asarray(
        calibrated_quality_628hc2c[
            "sensor_quality_risk"
        ],
        dtype=np.float64,
    )


    expected_branch_values_628hc2c = (
        np.asarray(
            calibrated_quality_628hc2c[
                expected_branch
            ],
            dtype=np.float64,
        )
    )


    # ------------------------------------------------------------
    # F. Numerical integrity
    # ------------------------------------------------------------

    if final_risk_628hc2c.shape != (
        N_WINDOWS_628HC2C,
    ):
        raise RuntimeError(
            f"{scenario_id}: bad final risk shape "
            f"{final_risk_628hc2c.shape}"
        )

    if not np.isfinite(
        final_risk_628hc2c
    ).all():
        raise RuntimeError(
            f"{scenario_id}: final risk not finite."
        )

    if (
        final_risk_628hc2c.min() < 0.0
        or final_risk_628hc2c.max() > 1.0
    ):
        raise RuntimeError(
            f"{scenario_id}: final risk outside [0,1]."
        )


    # ------------------------------------------------------------
    # G. Store full per-window risk
    # ------------------------------------------------------------

    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C[
        i,
        :
    ] = final_risk_628hc2c


    # ------------------------------------------------------------
    # H. Scenario-level summary
    # ------------------------------------------------------------

    scenario_median = float(
        np.median(
            final_risk_628hc2c
        )
    )

    scenario_mean = float(
        np.mean(
            final_risk_628hc2c
        )
    )

    scenario_q95 = float(
        np.quantile(
            final_risk_628hc2c,
            0.95,
            method="linear",
        )
    )

    fraction_above_q95 = float(
        np.mean(
            final_risk_628hc2c
            > FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
        )
    )


    scenario_rows_628hc2c.append(
        {
            "scenario_id": scenario_id,

            "scenario_number": int(
                row["scenario_number"]
            ),

            "family": family,

            "sensor": sensor,

            "severity": severity,

            "direction": (
                float(row["direction"])
                if pd.notna(row["direction"])
                else np.nan
            ),

            "alpha": (
                float(row["alpha"])
                if pd.notna(row["alpha"])
                else np.nan
            ),

            "requested_fraction": (
                float(
                    row["requested_fraction"]
                )
                if pd.notna(
                    row["requested_fraction"]
                )
                else np.nan
            ),

            "expected_branch": expected_branch,

            "expected_branch_median": float(
                np.median(
                    expected_branch_values_628hc2c
                )
            ),

            "expected_branch_mean": float(
                np.mean(
                    expected_branch_values_628hc2c
                )
            ),

            "sensor_quality_median": (
                scenario_median
            ),

            "sensor_quality_mean": (
                scenario_mean
            ),

            "sensor_quality_q95": (
                scenario_q95
            ),

            "fraction_above_fit_q95": (
                fraction_above_q95
            ),

            "median_shift_vs_clean": float(
                scenario_median
                - np.median(
                    clean_quality_628hc2c
                )
            ),

            "mean_shift_vs_clean": float(
                scenario_mean
                - np.mean(
                    clean_quality_628hc2c
                )
            ),
        }
    )


    # ------------------------------------------------------------
    # I. Progress
    # ------------------------------------------------------------

    elapsed = (
        time.perf_counter()
        - start_628hc2c
    )

    completed = i + 1

    rate = elapsed / completed

    remaining = (
        N_SCENARIOS_628HC2C
        - completed
    ) * rate


    if (
        completed == 1
        or completed % 15 == 0
        or completed == N_SCENARIOS_628HC2C
    ):

        print(
            f"[{completed:03d}/315] "
            f"{scenario_id} | "
            f"{family:<7} | "
            f"{sensor:<9} | "
            f"{severity:<6} | "
            f"median={scenario_median:.4f} | "
            f"mean={scenario_mean:.4f} | "
            f">q95={fraction_above_q95:.3f} | "
            f"elapsed={elapsed:.1f}s | "
            f"ETA≈{remaining / 60.0:.1f} min"
        )


    # ------------------------------------------------------------
    # J. Release large temporary objects
    # ------------------------------------------------------------

    del corrupted_raw_628hc2c
    del corrupted_normalized_628hc2c
    del X_corrupted_628hc2c
    del raw_quality_628hc2c
    del calibrated_quality_628hc2c
    del final_risk_628hc2c
    del expected_branch_values_628hc2c

    if completed % 25 == 0:
        gc.collect()


total_runtime_628hc2c = (
    time.perf_counter()
    - start_628hc2c
)


# ================================================================
# 8. CREATE FULL SCENARIO RESPONSE TABLE
# ================================================================

FD002_SENSOR_QUALITY_RESPONSE_628HC2C = (
    pd.DataFrame(
        scenario_rows_628hc2c
    )
)


print(
    "\n--- FULL SENSOR-QUALITY RESPONSE CREATED ---"
)

print(
    "Scenario rows :",
    len(
        FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    ),
)

print(
    "Risk matrix   :",
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C.shape,
)

print(
    "Runtime (sec) :",
    total_runtime_628hc2c,
)

print(
    "Runtime (min) :",
    total_runtime_628hc2c / 60.0,
)


# ================================================================
# 9. GLOBAL MATRIX INTEGRITY
# ================================================================

matrix_shape_ok_628hc2c = bool(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C.shape
    == (
        N_SCENARIOS_628HC2C,
        N_WINDOWS_628HC2C,
    )
)

matrix_finite_628hc2c = bool(
    np.isfinite(
        FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C
    ).all()
)

matrix_range_ok_628hc2c = bool(
    (
        FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C
        >= 0.0
    ).all()
    and
    (
        FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C
        <= 1.0
    ).all()
)


if not matrix_shape_ok_628hc2c:
    raise RuntimeError(
        "Full sensor-quality matrix has wrong shape."
    )

if not matrix_finite_628hc2c:
    raise RuntimeError(
        "Full sensor-quality matrix contains "
        "non-finite values."
    )

if not matrix_range_ok_628hc2c:
    raise RuntimeError(
        "Full sensor-quality matrix contains "
        "values outside [0,1]."
    )


# ================================================================
# 10. VERIFY CLEAN SOURCE UNCHANGED
# ================================================================

clean_source_after_628hc2c = (
    dev_base_628hc2c[
        [
            "unit_id",
            "cycle",
            *model_sensors_628hc2c,
        ]
    ]
    .to_numpy(copy=True)
)


clean_source_unchanged_628hc2c = bool(
    np.array_equal(
        clean_source_before_628hc2c,
        clean_source_after_628hc2c,
        equal_nan=True,
    )
)


if not clean_source_unchanged_628hc2c:
    raise RuntimeError(
        "Clean DEV corruption source was modified."
    )


# ================================================================
# 11. FAMILY × SEVERITY RESPONSE
# ================================================================

FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C = (
    FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        sensors=(
            "sensor",
            "nunique",
        ),

        median_expected_branch=(
            "expected_branch_median",
            "median",
        ),

        mean_expected_branch=(
            "expected_branch_mean",
            "mean",
        ),

        median_sensor_quality=(
            "sensor_quality_median",
            "median",
        ),

        mean_sensor_quality=(
            "sensor_quality_mean",
            "mean",
        ),

        median_q95=(
            "sensor_quality_q95",
            "median",
        ),

        mean_fraction_above_fit_q95=(
            "fraction_above_fit_q95",
            "mean",
        ),

        median_shift_vs_clean=(
            "median_shift_vs_clean",
            "median",
        ),

        mean_shift_vs_clean=(
            "mean_shift_vs_clean",
            "mean",
        ),
    )
)


family_order_628hc2c = pd.CategoricalDtype(
    categories=[
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ],
    ordered=True,
)

severity_order_628hc2c = pd.CategoricalDtype(
    categories=[
        "low",
        "medium",
        "high",
    ],
    ordered=True,
)


FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C[
    "family"
] = (
    FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C[
        "family"
    ]
    .astype(
        family_order_628hc2c
    )
)

FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C[
    "severity"
] = (
    FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C[
        "severity"
    ]
    .astype(
        severity_order_628hc2c
    )
)


FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C = (
    FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C
    .sort_values(
        [
            "family",
            "severity",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n--- FAMILY × SEVERITY FULL SENSOR-QUALITY RESPONSE ---"
)

display(
    FD002_SENSOR_QUALITY_FAMILY_SEVERITY_628HC2C
)


# ================================================================
# 12. SENSOR-LEVEL RESPONSE
# ================================================================

FD002_SENSOR_QUALITY_SENSOR_RESPONSE_628HC2C = (
    FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    .groupby(
        [
            "family",
            "sensor",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        median_quality=(
            "sensor_quality_median",
            "median",
        ),

        mean_quality=(
            "sensor_quality_mean",
            "mean",
        ),

        mean_fraction_above_fit_q95=(
            "fraction_above_fit_q95",
            "mean",
        ),

        maximum_quality_median=(
            "sensor_quality_median",
            "max",
        ),
    )
)


# ================================================================
# 13. TOP 20 STRONGEST QUALITY RESPONSES
# ================================================================

print(
    "\n--- TOP 20 SCENARIOS BY SENSOR-QUALITY MEDIAN ---"
)

display(
    FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    .sort_values(
        "sensor_quality_median",
        ascending=False,
    )
    .head(20)[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "direction",
            "sensor_quality_median",
            "sensor_quality_mean",
            "fraction_above_fit_q95",
            "median_shift_vs_clean",
        ]
    ]
)


# ================================================================
# 14. LOWEST 20 QUALITY RESPONSES
# ================================================================

print(
    "\n--- LOWEST 20 SCENARIOS BY SENSOR-QUALITY MEDIAN ---"
)

display(
    FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    .sort_values(
        "sensor_quality_median",
        ascending=True,
    )
    .head(20)[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "direction",
            "sensor_quality_median",
            "sensor_quality_mean",
            "fraction_above_fit_q95",
            "median_shift_vs_clean",
        ]
    ]
)


# ================================================================
# 15. SEVERITY MONOTONICITY DIAGNOSTIC
# ================================================================
#
# This is descriptive only.
# It does NOT impose a requirement that every sensor/family must be
# perfectly monotonic, because model-free corruption detectability
# can differ by sensor.
# ================================================================

severity_rank_628hc2c = {
    "low": 0,
    "medium": 1,
    "high": 2,
}


monotonic_rows_628hc2c = []

for (
    family,
    sensor,
), group in (
    FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    .groupby(
        [
            "family",
            "sensor",
        ],
        sort=False,
    )
):

    # Bias and drift have ± directions.
    # First collapse directions within each severity.
    collapsed = (
        group
        .groupby(
            "severity",
            as_index=False,
        )
        .agg(
            median_quality=(
                "sensor_quality_median",
                "median",
            ),

            mean_exceedance=(
                "fraction_above_fit_q95",
                "mean",
            ),
        )
    )

    collapsed["severity_rank"] = (
        collapsed[
            "severity"
        ]
        .map(
            severity_rank_628hc2c
        )
    )

    collapsed = (
        collapsed
        .sort_values(
            "severity_rank"
        )
    )

    quality_values = (
        collapsed[
            "median_quality"
        ]
        .to_numpy(dtype=float)
    )

    exceedance_values = (
        collapsed[
            "mean_exceedance"
        ]
        .to_numpy(dtype=float)
    )

    quality_monotonic = bool(
        len(quality_values) == 3
        and np.all(
            np.diff(
                quality_values
            ) >= -1e-12
        )
    )

    exceedance_monotonic = bool(
        len(exceedance_values) == 3
        and np.all(
            np.diff(
                exceedance_values
            ) >= -1e-12
        )
    )

    monotonic_rows_628hc2c.append(
        {
            "family": family,
            "sensor": sensor,
            "quality_median_monotonic":
                quality_monotonic,
            "q95_exceedance_monotonic":
                exceedance_monotonic,
        }
    )


FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C = (
    pd.DataFrame(
        monotonic_rows_628hc2c
    )
)


print(
    "\n--- SEVERITY MONOTONICITY DIAGNOSTIC ---"
)

print(
    "Family-sensor groups:",
    len(
        FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C
    ),
)

print(
    "Median-quality monotonic:",
    int(
        FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C[
            "quality_median_monotonic"
        ].sum()
    ),
    "/",
    len(
        FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C
    ),
)

print(
    "q95-exceedance monotonic:",
    int(
        FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C[
            "q95_exceedance_monotonic"
        ].sum()
    ),
    "/",
    len(
        FD002_SENSOR_QUALITY_MONOTONICITY_628HC2C
    ),
)


# ================================================================
# 16. FINAL INTEGRITY AUDIT
# ================================================================

response_rows_ok_628hc2c = bool(
    len(
        FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    )
    == 315
)

response_ids_exact_628hc2c = bool(
    np.array_equal(
        FD002_SENSOR_QUALITY_RESPONSE_628HC2C[
            "scenario_id"
        ].astype(str).to_numpy(),
        expected_ids_628hc2c,
    )
)


FD002_SENSOR_QUALITY_AUDIT_628HC2C = (
    pd.DataFrame(
        [
            {
                "check": "scenario_count_315",
                "passed":
                    response_rows_ok_628hc2c,
            },

            {
                "check": "scenario_order_exact",
                "passed":
                    response_ids_exact_628hc2c,
            },

            {
                "check": "matrix_shape_exact",
                "passed":
                    matrix_shape_ok_628hc2c,
            },

            {
                "check": "matrix_finite",
                "passed":
                    matrix_finite_628hc2c,
            },

            {
                "check": "matrix_within_0_1",
                "passed":
                    matrix_range_ok_628hc2c,
            },

            {
                "check": "clean_source_unchanged",
                "passed":
                    clean_source_unchanged_628hc2c,
            },
        ]
    )
)


print(
    "\n--- FINAL SENSOR-QUALITY AUDIT ---"
)

display(
    FD002_SENSOR_QUALITY_AUDIT_628HC2C
)


if not bool(
    FD002_SENSOR_QUALITY_AUDIT_628HC2C[
        "passed"
    ].all()
):
    raise RuntimeError(
        "One or more Step 6.28H-C2-C "
        "integrity checks failed."
    )


# ================================================================
# 17. FREEZE CONTRACT
# ================================================================

FD002_SENSOR_QUALITY_FULL_CONTRACT_628HC2C = {
    "dataset": "FD002",

    "scenarios": 315,

    "dev_windows_per_scenario": 8367,

    "risk_matrix_shape":
        FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C.shape,

    "calibration_source":
        "FD002_FIT_only",

    "sensor_quality_recipe":
        "Step_6.28H-C2-B_frozen",

    "fit_q95_diagnostic_marker":
        FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B,

    "scenario_order_exact":
        scenario_id_exact_628hc2c,

    "risk_matrix_finite":
        matrix_finite_628hc2c,

    "risk_matrix_within_0_1":
        matrix_range_ok_628hc2c,

    "clean_source_unchanged":
        clean_source_unchanged_628hc2c,

    "dev_rul_labels_used": False,

    "official_test_labels_read": False,

    "test_scored": False,

    "abstention_threshold_selected": False,

    "model_executed": False,

    "model_retrained": False,

    "files_written": False,
}


# ================================================================
# 18. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 74)
print("STEP 6.28H-C2-C FINAL SUMMARY")
print("=" * 74)

print(
    "Scenarios executed              :",
    len(
        FD002_SENSOR_QUALITY_RESPONSE_628HC2C
    ),
)

print(
    "DEV windows/scenario            :",
    N_WINDOWS_628HC2C,
)

print(
    "Sensor-quality risk matrix      :",
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C.shape,
)

print(
    "Risk matrix finite              :",
    matrix_finite_628hc2c,
)

print(
    "Risk values within [0,1]        :",
    matrix_range_ok_628hc2c,
)

print(
    "Clean DEV baseline median       :",
    float(
        np.median(
            clean_quality_628hc2c
        )
    ),
)

print(
    "Clean DEV baseline mean         :",
    float(
        np.mean(
            clean_quality_628hc2c
        )
    ),
)

print(
    "FIT q95 diagnostic marker       :",
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B,
)

print(
    "Clean source unchanged          :",
    clean_source_unchanged_628hc2c,
)

print(
    "Total runtime (sec)             :",
    total_runtime_628hc2c,
)

print(
    "Total runtime (min)             :",
    total_runtime_628hc2c / 60.0,
)

print(
    "DEV RUL labels used             :",
    False,
)

print(
    "Official TEST labels read       :",
    False,
)

print(
    "TEST scored                     :",
    False,
)

print(
    "Abstention threshold selected   :",
    False,
)

print(
    "Model executed                  :",
    False,
)

print(
    "Model retrained                 :",
    False,
)

print(
    "Files written                   :",
    False,
)

print(
    "\n=== END STEP 6.28H-C2-C ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C2-D — SENSOR-QUALITY-ONLY SELECTIVE RISK
# ================================================================
#
# PURPOSE
# -------
# Evaluate whether the frozen FIT-calibrated, label-free
# sensor-quality risk can identify DEV windows on which corrupted
# RUL predictions are less reliable.
#
# This is the SENSOR-QUALITY-ONLY counterpart of the earlier
# uncertainty-only selective-risk experiment.
#
# IMPORTANT METHODOLOGICAL DETAIL
# -------------------------------
# Some corruption scenarios (especially dropout / severe stuck)
# contain many identical risk values, often exactly 1.0.
#
# Therefore this step uses a TIE-AWARE expected selective MAE.
# At a requested coverage, if the acceptance boundary falls inside
# a group of samples having identical risk, the expected error of
# that tied group is used proportionally.
#
# This avoids arbitrary row-order tie breaking.
#
# NO:
#   * official TEST labels
#   * TEST scoring
#   * abstention threshold selection
#   * model execution
#   * model retraining
#   * file writing
#
# DEV RUL labels ARE used here because this is a DEV diagnostic
# evaluating whether the risk signal ranks prediction errors.
# ================================================================

import time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


print("=" * 76)
print("STEP 6.28H-C2-D — SENSOR-QUALITY-ONLY SELECTIVE RISK")
print("=" * 76)


# ================================================================
# 1. REQUIRED OBJECTS
# ================================================================

required_objects_628hc2d = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc2d = []

for name in required_objects_628hc2d:
    exists = name in globals()

    print(
        f"{name:<58}: {exists}"
    )

    if not exists:
        missing_628hc2d.append(name)


if missing_628hc2d:
    raise RuntimeError(
        "Missing required Step 6.28H-C2-D objects:\n"
        + "\n".join(
            missing_628hc2d
        )
    )


# ================================================================
# 2. FREEZE INPUT ARRAYS
# ================================================================

manifest_628hc2d = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)


prediction_matrix_628hc2d = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64,
)


quality_risk_matrix_628hc2d = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)


y_dev_628hc2d = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64,
).reshape(-1)


clean_predictions_628hc2d = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
).reshape(-1)


# ================================================================
# 3. ROBUSTLY EXTRACT CLEAN SENSOR-QUALITY RISK
# ================================================================

clean_quality_object_628hc2d = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)


if isinstance(
    clean_quality_object_628hc2d,
    pd.DataFrame,
):
    if (
        "sensor_quality_risk"
        not in
        clean_quality_object_628hc2d.columns
    ):
        raise RuntimeError(
            "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
            "does not contain sensor_quality_risk."
        )

    clean_quality_risk_628hc2d = np.asarray(
        clean_quality_object_628hc2d[
            "sensor_quality_risk"
        ],
        dtype=np.float64,
    )

elif isinstance(
    clean_quality_object_628hc2d,
    dict,
):
    if (
        "sensor_quality_risk"
        not in
        clean_quality_object_628hc2d
    ):
        raise RuntimeError(
            "Clean sensor-quality dictionary does not "
            "contain sensor_quality_risk."
        )

    clean_quality_risk_628hc2d = np.asarray(
        clean_quality_object_628hc2d[
            "sensor_quality_risk"
        ],
        dtype=np.float64,
    )

else:
    clean_quality_risk_628hc2d = np.asarray(
        clean_quality_object_628hc2d,
        dtype=np.float64,
    )


clean_quality_risk_628hc2d = (
    clean_quality_risk_628hc2d
    .reshape(-1)
)


# ================================================================
# 4. POPULATION CONTRACT
# ================================================================

N_SCENARIOS_628HC2D = 315
N_WINDOWS_628HC2D = 8367


print("\n--- POPULATION CONTRACT ---")

print(
    "Scenarios              :",
    len(manifest_628hc2d),
)

print(
    "Prediction matrix      :",
    prediction_matrix_628hc2d.shape,
)

print(
    "Sensor-quality matrix  :",
    quality_risk_matrix_628hc2d.shape,
)

print(
    "DEV target             :",
    y_dev_628hc2d.shape,
)

print(
    "Clean predictions      :",
    clean_predictions_628hc2d.shape,
)

print(
    "Clean quality risk     :",
    clean_quality_risk_628hc2d.shape,
)


expected_matrix_shape_628hc2d = (
    N_SCENARIOS_628HC2D,
    N_WINDOWS_628HC2D,
)


if prediction_matrix_628hc2d.shape != (
    expected_matrix_shape_628hc2d
):
    raise RuntimeError(
        "Unexpected corrupted prediction matrix shape: "
        f"{prediction_matrix_628hc2d.shape}"
    )


if quality_risk_matrix_628hc2d.shape != (
    expected_matrix_shape_628hc2d
):
    raise RuntimeError(
        "Unexpected sensor-quality matrix shape: "
        f"{quality_risk_matrix_628hc2d.shape}"
    )


if y_dev_628hc2d.shape != (
    N_WINDOWS_628HC2D,
):
    raise RuntimeError(
        "Unexpected DEV target shape."
    )


if clean_predictions_628hc2d.shape != (
    N_WINDOWS_628HC2D,
):
    raise RuntimeError(
        "Unexpected clean prediction shape."
    )


if clean_quality_risk_628hc2d.shape != (
    N_WINDOWS_628HC2D,
):
    raise RuntimeError(
        "Unexpected clean sensor-quality risk shape."
    )


# ================================================================
# 5. NUMERICAL INTEGRITY
# ================================================================

objects_to_check_628hc2d = {
    "prediction_matrix":
        prediction_matrix_628hc2d,

    "quality_risk_matrix":
        quality_risk_matrix_628hc2d,

    "y_dev":
        y_dev_628hc2d,

    "clean_predictions":
        clean_predictions_628hc2d,

    "clean_quality_risk":
        clean_quality_risk_628hc2d,
}


print("\n--- NUMERICAL INTEGRITY ---")

for (
    name,
    values,
) in objects_to_check_628hc2d.items():

    finite = bool(
        np.isfinite(
            values
        ).all()
    )

    print(
        f"{name:<24}: finite={finite}"
    )

    if not finite:
        raise RuntimeError(
            f"{name} contains non-finite values."
        )


risk_range_ok_628hc2d = bool(
    (
        quality_risk_matrix_628hc2d
        >= 0.0
    ).all()
    and
    (
        quality_risk_matrix_628hc2d
        <= 1.0
    ).all()
)


clean_risk_range_ok_628hc2d = bool(
    (
        clean_quality_risk_628hc2d
        >= 0.0
    ).all()
    and
    (
        clean_quality_risk_628hc2d
        <= 1.0
    ).all()
)


print(
    "Corrupted quality risk within [0,1] :",
    risk_range_ok_628hc2d,
)

print(
    "Clean quality risk within [0,1]     :",
    clean_risk_range_ok_628hc2d,
)


if not risk_range_ok_628hc2d:
    raise RuntimeError(
        "Corrupted sensor-quality risk outside [0,1]."
    )

if not clean_risk_range_ok_628hc2d:
    raise RuntimeError(
        "Clean sensor-quality risk outside [0,1]."
    )


# ================================================================
# 6. HELPER — SAFE SPEARMAN
# ================================================================

def safe_spearman_628hc2d(
    risk,
    error,
):
    """
    Spearman rank correlation between risk and absolute error.

    Returns NaN when the risk vector is constant, because no
    ranking information exists in that case.
    """

    risk = np.asarray(
        risk,
        dtype=np.float64,
    ).reshape(-1)

    error = np.asarray(
        error,
        dtype=np.float64,
    ).reshape(-1)


    if risk.shape != error.shape:
        raise ValueError(
            "Risk and error vectors must have equal shape."
        )


    if (
        not np.isfinite(risk).all()
        or
        not np.isfinite(error).all()
    ):
        raise ValueError(
            "Risk/error vectors must be finite."
        )


    if np.all(
        risk == risk[0]
    ):
        return np.nan


    rho = spearmanr(
        risk,
        error,
    ).statistic


    return float(rho)


# ================================================================
# 7. HELPER — TIE-AWARE EXPECTED SELECTIVE MAE
# ================================================================

def tie_aware_selective_mae_628hc2d(
    absolute_error,
    risk,
    coverage,
):
    """
    Compute expected MAE at requested coverage while treating
    equal-risk boundary samples fairly.

    Lower risk = accepted first.

    If the coverage boundary cuts through a tied risk group,
    proportional expected error mass from that entire tied group
    is used instead of arbitrary row-order selection.

    Example:
        if all risk values are identical,
        MAE at every coverage == full MAE.

    This is intentional and prevents artificial gains from ties.
    """

    error = np.asarray(
        absolute_error,
        dtype=np.float64,
    ).reshape(-1)

    risk = np.asarray(
        risk,
        dtype=np.float64,
    ).reshape(-1)


    if error.shape != risk.shape:
        raise ValueError(
            "Error and risk shapes differ."
        )


    if not (
        0.0 < float(coverage) <= 1.0
    ):
        raise ValueError(
            "Coverage must be in (0,1]."
        )


    n = len(error)

    target_mass = (
        float(coverage)
        * float(n)
    )


    if target_mass <= 0.0:
        raise ValueError(
            "Requested accepted mass is zero."
        )


    # Sort by ascending risk.
    order = np.argsort(
        risk,
        kind="mergesort",
    )


    sorted_risk = risk[
        order
    ]

    sorted_error = error[
        order
    ]


    # Identify exact tied groups.
    (
        unique_risk,
        start_indices,
        group_counts,
    ) = np.unique(
        sorted_risk,
        return_index=True,
        return_counts=True,
    )


    accepted_mass = 0.0
    accepted_error_mass = 0.0


    for (
        start,
        count,
    ) in zip(
        start_indices,
        group_counts,
    ):

        count = int(
            count
        )

        group_error = (
            sorted_error[
                start:
                start + count
            ]
        )

        group_mean_error = float(
            np.mean(
                group_error
            )
        )


        remaining_mass = (
            target_mass
            - accepted_mass
        )


        if remaining_mass <= 1e-15:
            break


        if count <= (
            remaining_mass
            + 1e-12
        ):

            accepted_error_mass += float(
                np.sum(
                    group_error
                )
            )

            accepted_mass += float(
                count
            )

        else:

            # Boundary falls within a tied group.
            #
            # Use expected contribution from a random member of
            # the tied group instead of arbitrary index ordering.

            accepted_error_mass += (
                remaining_mass
                * group_mean_error
            )

            accepted_mass += (
                remaining_mass
            )

            break


    if not np.isclose(
        accepted_mass,
        target_mass,
        rtol=0.0,
        atol=1e-8,
    ):
        raise RuntimeError(
            "Tie-aware coverage mass was not achieved."
        )


    selective_mae = (
        accepted_error_mass
        / target_mass
    )


    return float(
        selective_mae
    )


# ================================================================
# 8. HELPER — TIE DIAGNOSTICS
# ================================================================

def risk_tie_diagnostics_628hc2d(
    risk,
):
    risk = np.asarray(
        risk,
        dtype=np.float64,
    ).reshape(-1)

    (
        unique_values,
        counts,
    ) = np.unique(
        risk,
        return_counts=True,
    )

    return {
        "risk_unique_values":
            int(
                len(
                    unique_values
                )
            ),

        "largest_tie_count":
            int(
                counts.max()
            ),

        "largest_tie_fraction":
            float(
                counts.max()
                / len(risk)
            ),

        "risk_constant":
            bool(
                len(
                    unique_values
                )
                == 1
            ),
    }


# ================================================================
# 9. FIXED DIAGNOSTIC COVERAGES
# ================================================================

COVERAGES_628HC2D = (
    0.90,
    0.80,
    0.70,
)


print(
    "\n--- SELECTIVE-RISK EVALUATION POLICY ---"
)

print(
    "Risk direction       : lower risk accepted first"
)

print(
    "Coverages            :",
    COVERAGES_628HC2D,
)

print(
    "Boundary tie policy  : expected error within tied risk group"
)

print(
    "Threshold selection  : False"
)


# ================================================================
# 10. CLEAN DEV SENSOR-QUALITY DIAGNOSTIC
# ================================================================

clean_abs_error_628hc2d = np.abs(
    clean_predictions_628hc2d
    - y_dev_628hc2d
)


clean_full_mae_628hc2d = float(
    np.mean(
        clean_abs_error_628hc2d
    )
)


clean_rho_628hc2d = (
    safe_spearman_628hc2d(
        clean_quality_risk_628hc2d,
        clean_abs_error_628hc2d,
    )
)


clean_ties_628hc2d = (
    risk_tie_diagnostics_628hc2d(
        clean_quality_risk_628hc2d
    )
)


clean_selective_628hc2d = {}


for coverage in COVERAGES_628HC2D:

    mae_value = (
        tie_aware_selective_mae_628hc2d(
            clean_abs_error_628hc2d,
            clean_quality_risk_628hc2d,
            coverage,
        )
    )

    clean_selective_628hc2d[
        coverage
    ] = mae_value


print(
    "\n--- CLEAN SENSOR-QUALITY-ONLY DIAGNOSTIC ---"
)

print(
    "Clean MAE:",
    clean_full_mae_628hc2d,
)

print(
    "Risk/error Spearman:",
    clean_rho_628hc2d,
)

print(
    "Risk unique values:",
    clean_ties_628hc2d[
        "risk_unique_values"
    ],
)

print(
    "Largest tie fraction:",
    clean_ties_628hc2d[
        "largest_tie_fraction"
    ],
)


for coverage in COVERAGES_628HC2D:

    selective_mae = (
        clean_selective_628hc2d[
            coverage
        ]
    )

    gain = (
        clean_full_mae_628hc2d
        - selective_mae
    )

    print(
        f"MAE @ {int(coverage * 100)}% coverage:"
        f" {selective_mae}"
        f" | gain={gain:+.6f}"
    )


# ================================================================
# 11. EVALUATE ALL 315 CORRUPTED SCENARIOS
# ================================================================

print(
    "\n--- APPLYING SENSOR-QUALITY RISK TO "
    "315 CORRUPTED SCENARIOS ---"
)


scenario_results_628hc2d = []

start_628hc2d = (
    time.perf_counter()
)


for i, row in (
    manifest_628hc2d.iterrows()
):

    scenario_id = str(
        row["scenario_id"]
    )

    prediction = (
        prediction_matrix_628hc2d[
            i
        ]
    )

    risk = (
        quality_risk_matrix_628hc2d[
            i
        ]
    )


    absolute_error = np.abs(
        prediction
        - y_dev_628hc2d
    )


    full_mae = float(
        np.mean(
            absolute_error
        )
    )


    rho = (
        safe_spearman_628hc2d(
            risk,
            absolute_error,
        )
    )


    tie_info = (
        risk_tie_diagnostics_628hc2d(
            risk
        )
    )


    result = {
        "scenario_id":
            scenario_id,

        "scenario_number":
            int(
                row["scenario_number"]
            ),

        "family":
            str(
                row["family"]
            ),

        "sensor":
            str(
                row["sensor"]
            ),

        "severity":
            str(
                row["severity"]
            ),

        "direction":
            (
                float(
                    row["direction"]
                )
                if pd.notna(
                    row["direction"]
                )
                else np.nan
            ),

        "full_mae":
            full_mae,

        "risk_error_spearman":
            rho,

        "risk_median":
            float(
                np.median(
                    risk
                )
            ),

        "risk_mean":
            float(
                np.mean(
                    risk
                )
            ),

        **tie_info,
    }


    for coverage in (
        COVERAGES_628HC2D
    ):

        pct = int(
            round(
                coverage
                * 100
            )
        )

        selective_mae = (
            tie_aware_selective_mae_628hc2d(
                absolute_error,
                risk,
                coverage,
            )
        )


        gain = (
            full_mae
            - selective_mae
        )


        result[
            f"mae_at_{pct}pct_coverage"
        ] = selective_mae

        result[
            f"mae_gain_at_{pct}pct_coverage"
        ] = gain


    scenario_results_628hc2d.append(
        result
    )


    completed = (
        i + 1
    )


    if (
        completed == 1
        or
        completed % 50 == 0
        or
        completed == 315
    ):

        elapsed = (
            time.perf_counter()
            - start_628hc2d
        )

        gain80 = result[
            "mae_gain_at_80pct_coverage"
        ]

        rho_text = (
            "nan"
            if np.isnan(rho)
            else f"{rho:.4f}"
        )

        print(
            f"[{completed:03d}/315] "
            f"{scenario_id} | "
            f"{str(row['family']):<7} | "
            f"{str(row['severity']):<6} | "
            f"MAE={full_mae:.4f} | "
            f"Spearman={rho_text} | "
            f"gain@80={gain80:+.4f} | "
            f"largest_tie="
            f"{tie_info['largest_tie_fraction']:.3f} | "
            f"elapsed={elapsed:.1f}s"
        )


total_runtime_628hc2d = (
    time.perf_counter()
    - start_628hc2d
)


# ================================================================
# 12. FULL RESPONSE TABLE
# ================================================================

FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D = (
    pd.DataFrame(
        scenario_results_628hc2d
    )
)


print(
    "\n--- FULL SENSOR-QUALITY SELECTIVE RESPONSE CREATED ---"
)

print(
    "Rows:",
    len(
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    ),
)

print(
    "Runtime (sec):",
    total_runtime_628hc2d,
)


# ================================================================
# 13. FAMILY × SEVERITY SUMMARY
# ================================================================

FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D = (
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        median_full_mae=(
            "full_mae",
            "median",
        ),

        median_risk_error_spearman=(
            "risk_error_spearman",
            "median",
        ),

        mean_risk_error_spearman=(
            "risk_error_spearman",
            "mean",
        ),

        median_gain_90=(
            "mae_gain_at_90pct_coverage",
            "median",
        ),

        median_gain_80=(
            "mae_gain_at_80pct_coverage",
            "median",
        ),

        median_gain_70=(
            "mae_gain_at_70pct_coverage",
            "median",
        ),

        mean_gain_80=(
            "mae_gain_at_80pct_coverage",
            "mean",
        ),

        median_risk=(
            "risk_median",
            "median",
        ),

        median_largest_tie_fraction=(
            "largest_tie_fraction",
            "median",
        ),

        constant_risk_scenarios=(
            "risk_constant",
            "sum",
        ),
    )
)


family_order_628hc2d = pd.CategoricalDtype(
    categories=[
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ],
    ordered=True,
)


severity_order_628hc2d = pd.CategoricalDtype(
    categories=[
        "low",
        "medium",
        "high",
    ],
    ordered=True,
)


FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D[
    "family"
] = (
    FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D[
        "family"
    ]
    .astype(
        family_order_628hc2d
    )
)


FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D[
    "severity"
] = (
    FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D[
        "severity"
    ]
    .astype(
        severity_order_628hc2d
    )
)


FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D = (
    FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D
    .sort_values(
        [
            "family",
            "severity",
        ]
    )
    .reset_index(drop=True)
)


print(
    "\n--- FAMILY × SEVERITY: "
    "SENSOR-QUALITY-ONLY SELECTIVE RESPONSE ---"
)

display(
    FD002_SENSOR_QUALITY_SELECTIVE_FAMILY_SEVERITY_628HC2D
)


# ================================================================
# 14. CONSTANT / SATURATED RISK DIAGNOSTIC
# ================================================================

constant_cases_628hc2d = (
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
            "risk_constant"
        ]
    ]
    .copy()
)


print(
    "\n--- CONSTANT-RISK SCENARIOS ---"
)

print(
    "Count:",
    len(
        constant_cases_628hc2d
    ),
)


if len(
    constant_cases_628hc2d
) > 0:

    display(
        constant_cases_628hc2d[
            [
                "scenario_id",
                "family",
                "sensor",
                "severity",
                "risk_median",
                "full_mae",
                "mae_gain_at_80pct_coverage",
            ]
        ]
    )


# ================================================================
# 15. WORST 15 CASES AT 80% COVERAGE
# ================================================================

print(
    "\n--- WORST 15 CASES AT 80% COVERAGE ---"
)

display(
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    .sort_values(
        "mae_gain_at_80pct_coverage",
        ascending=True,
    )
    .head(15)[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "full_mae",
            "risk_error_spearman",
            "mae_at_80pct_coverage",
            "mae_gain_at_80pct_coverage",
            "risk_median",
            "largest_tie_fraction",
        ]
    ]
)


# ================================================================
# 16. BEST 15 CASES AT 80% COVERAGE
# ================================================================

print(
    "\n--- BEST 15 CASES AT 80% COVERAGE ---"
)

display(
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    .sort_values(
        "mae_gain_at_80pct_coverage",
        ascending=False,
    )
    .head(15)[
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "full_mae",
            "risk_error_spearman",
            "mae_at_80pct_coverage",
            "mae_gain_at_80pct_coverage",
            "risk_median",
            "largest_tie_fraction",
        ]
    ]
)


# ================================================================
# 17. GLOBAL DIAGNOSTICS
# ================================================================

gain80_628hc2d = (
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
        "mae_gain_at_80pct_coverage"
    ]
    .to_numpy(
        dtype=float
    )
)


rho_values_628hc2d = (
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
        "risk_error_spearman"
    ]
    .to_numpy(
        dtype=float
    )
)


finite_rho_mask_628hc2d = (
    np.isfinite(
        rho_values_628hc2d
    )
)


positive_gain_count_628hc2d = int(
    np.sum(
        gain80_628hc2d
        > 1e-12
    )
)


negative_gain_count_628hc2d = int(
    np.sum(
        gain80_628hc2d
        < -1e-12
    )
)


zero_gain_count_628hc2d = int(
    len(
        gain80_628hc2d
    )
    - positive_gain_count_628hc2d
    - negative_gain_count_628hc2d
)


constant_risk_count_628hc2d = int(
    FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
        "risk_constant"
    ]
    .sum()
)


# ================================================================
# 18. FINAL INTEGRITY AUDIT
# ================================================================

response_count_ok_628hc2d = bool(
    len(
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    )
    == 315
)


scenario_order_ok_628hc2d = bool(
    np.array_equal(
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
            "scenario_id"
        ]
        .astype(str)
        .to_numpy(),

        manifest_628hc2d[
            "scenario_id"
        ]
        .astype(str)
        .to_numpy(),
    )
)


finite_mae_fields_628hc2d = bool(
    np.isfinite(
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D[
            [
                "full_mae",
                "mae_at_90pct_coverage",
                "mae_at_80pct_coverage",
                "mae_at_70pct_coverage",
                "mae_gain_at_90pct_coverage",
                "mae_gain_at_80pct_coverage",
                "mae_gain_at_70pct_coverage",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()
)


FD002_SENSOR_QUALITY_SELECTIVE_AUDIT_628HC2D = (
    pd.DataFrame(
        [
            {
                "check":
                    "scenario_count_315",

                "passed":
                    response_count_ok_628hc2d,
            },

            {
                "check":
                    "scenario_order_exact",

                "passed":
                    scenario_order_ok_628hc2d,
            },

            {
                "check":
                    "risk_matrix_shape_exact",

                "passed":
                    (
                        quality_risk_matrix_628hc2d.shape
                        ==
                        (315, 8367)
                    ),
            },

            {
                "check":
                    "risk_matrix_finite",

                "passed":
                    bool(
                        np.isfinite(
                            quality_risk_matrix_628hc2d
                        ).all()
                    ),
            },

            {
                "check":
                    "risk_matrix_within_0_1",

                "passed":
                    risk_range_ok_628hc2d,
            },

            {
                "check":
                    "selective_mae_fields_finite",

                "passed":
                    finite_mae_fields_628hc2d,
            },
        ]
    )
)


print(
    "\n--- FINAL SELECTIVE-RISK AUDIT ---"
)

display(
    FD002_SENSOR_QUALITY_SELECTIVE_AUDIT_628HC2D
)


if not bool(
    FD002_SENSOR_QUALITY_SELECTIVE_AUDIT_628HC2D[
        "passed"
    ].all()
):
    raise RuntimeError(
        "Step 6.28H-C2-D integrity audit failed."
    )


# ================================================================
# 19. FREEZE CONTRACT
# ================================================================

FD002_SENSOR_QUALITY_SELECTIVE_CONTRACT_628HC2D = {
    "dataset":
        "FD002",

    "risk_source":
        "FIT-calibrated label-free sensor-quality risk",

    "scenarios":
        315,

    "dev_windows":
        8367,

    "evaluation_coverages":
        [0.90, 0.80, 0.70],

    "acceptance_direction":
        "lowest_risk_first",

    "tie_policy":
        "expected_error_within_boundary_tie_group",

    "risk_matrix_shape":
        tuple(
            quality_risk_matrix_628hc2d.shape
        ),

    "dev_rul_labels_used_for_diagnostic":
        True,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "abstention_threshold_selected":
        False,

    "model_executed":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ================================================================
# 20. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 76)
print("STEP 6.28H-C2-D FINAL SUMMARY")
print("=" * 76)

print(
    "Scenarios evaluated                 :",
    len(
        FD002_SENSOR_QUALITY_SELECTIVE_RESPONSE_628HC2D
    ),
)

print(
    "DEV windows/scenario                :",
    N_WINDOWS_628HC2D,
)

print(
    "Risk matrix shape                   :",
    quality_risk_matrix_628hc2d.shape,
)

print(
    "Tie-aware evaluation                :",
    True,
)

print(
    "Clean full MAE                      :",
    clean_full_mae_628hc2d,
)

print(
    "Clean risk/error Spearman           :",
    clean_rho_628hc2d,
)

print(
    "Clean gain @90% coverage            :",
    clean_full_mae_628hc2d
    - clean_selective_628hc2d[0.90],
)

print(
    "Clean gain @80% coverage            :",
    clean_full_mae_628hc2d
    - clean_selective_628hc2d[0.80],
)

print(
    "Clean gain @70% coverage            :",
    clean_full_mae_628hc2d
    - clean_selective_628hc2d[0.70],
)

print(
    "Median corrupted risk/error Spearman:",
    float(
        np.nanmedian(
            rho_values_628hc2d
        )
    ),
)

print(
    "Finite Spearman scenarios           :",
    int(
        finite_rho_mask_628hc2d.sum()
    ),
    "/ 315",
)

print(
    "Median MAE gain @80% coverage       :",
    float(
        np.median(
            gain80_628hc2d
        )
    ),
)

print(
    "Mean MAE gain @80% coverage         :",
    float(
        np.mean(
            gain80_628hc2d
        )
    ),
)

print(
    "Scenarios gain @80% > 0             :",
    positive_gain_count_628hc2d,
    "/ 315",
)

print(
    "Scenarios gain @80% = 0             :",
    zero_gain_count_628hc2d,
    "/ 315",
)

print(
    "Scenarios gain @80% < 0             :",
    negative_gain_count_628hc2d,
    "/ 315",
)

print(
    "Constant-risk scenarios             :",
    constant_risk_count_628hc2d,
)

print(
    "Runtime (sec)                       :",
    total_runtime_628hc2d,
)

print(
    "DEV RUL labels used                 :",
    True,
)

print(
    "Official TEST labels read           :",
    False,
)

print(
    "TEST scored                         :",
    False,
)

print(
    "Abstention threshold selected       :",
    False,
)

print(
    "Model executed                      :",
    False,
)

print(
    "Model retrained                     :",
    False,
)

print(
    "Files written                       :",
    False,
)

print(
    "\n=== END STEP 6.28H-C2-D ==="
)

In [ ]:
# ============================================================
# STEP 6.28H-C3-A0
# COMBINED-RELIABILITY COMPLEMENTARITY AUDIT — PRE-FLIGHT
#
# PURPOSE
# -------
# Establish the exact frozen objects required for the next
# complementarity analysis before constructing any combined
# reliability signal.
#
# IMPORTANT
# ---------
# - DEV only
# - No official TEST labels
# - No TEST scoring
# - No threshold selection
# - No model retraining
# - No corruption generation
# - No files written
# ============================================================

import numpy as np
import pandas as pd

print("=" * 78)
print("STEP 6.28H-C3-A0 — COMBINED-RELIABILITY COMPLEMENTARITY PRE-FLIGHT")
print("=" * 78)

# ------------------------------------------------------------
# 1. REQUIRED CORE OBJECTS
# ------------------------------------------------------------

required_objects_628hc3a0 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "Y_DEV_PRIMARY_FD002_628D",

    # Uncertainty branch
    "MC_STD_MATRIX_FD002_628HC1C",
    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",

    # Sensor-quality branch
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
]

print("\n--- REQUIRED CORE OBJECTS ---")

required_status_628hc3a0 = {}

for name_628hc3a0 in required_objects_628hc3a0:
    present_628hc3a0 = name_628hc3a0 in globals()
    required_status_628hc3a0[name_628hc3a0] = present_628hc3a0
    print(f"{name_628hc3a0:<55}: {present_628hc3a0}")

missing_628hc3a0 = [
    name
    for name, present in required_status_628hc3a0.items()
    if not present
]

if missing_628hc3a0:
    raise RuntimeError(
        "Missing required frozen objects for Step 6.28H-C3-A0:\n"
        + "\n".join(f"  - {name}" for name in missing_628hc3a0)
    )


# ------------------------------------------------------------
# 2. FROZEN POPULATION CONTRACT
# ------------------------------------------------------------

manifest_628hc3a0 = FD002_CORRUPTION_MANIFEST_628GD.copy()

prediction_matrix_628hc3a0 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

mc_std_matrix_628hc3a0 = np.asarray(
    MC_STD_MATRIX_FD002_628HC1C,
    dtype=np.float64
)

quality_risk_matrix_628hc3a0 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

y_dev_628hc3a0 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_uncertainty_risk_628hc3a0 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)

clean_quality_risk_628hc3a0 = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B,
    dtype=np.float64
).reshape(-1)

n_scenarios_628hc3a0 = len(manifest_628hc3a0)
n_dev_628hc3a0 = len(y_dev_628hc3a0)

print("\n--- FROZEN POPULATION CONTRACT ---")
print("Scenarios                         :", n_scenarios_628hc3a0)
print("DEV windows                       :", n_dev_628hc3a0)
print("Prediction matrix                 :", prediction_matrix_628hc3a0.shape)
print("MC-STD matrix                     :", mc_std_matrix_628hc3a0.shape)
print("Sensor-quality risk matrix        :", quality_risk_matrix_628hc3a0.shape)
print("Clean uncertainty-risk vector     :", clean_uncertainty_risk_628hc3a0.shape)
print("Clean sensor-quality-risk vector  :", clean_quality_risk_628hc3a0.shape)


# ------------------------------------------------------------
# 3. STRUCTURAL CONTRACTS
# ------------------------------------------------------------

expected_matrix_shape_628hc3a0 = (
    n_scenarios_628hc3a0,
    n_dev_628hc3a0,
)

structural_checks_628hc3a0 = {
    "scenario_count_315":
        n_scenarios_628hc3a0 == 315,

    "prediction_matrix_shape":
        prediction_matrix_628hc3a0.shape
        == expected_matrix_shape_628hc3a0,

    "mc_std_matrix_shape":
        mc_std_matrix_628hc3a0.shape
        == expected_matrix_shape_628hc3a0,

    "quality_risk_matrix_shape":
        quality_risk_matrix_628hc3a0.shape
        == expected_matrix_shape_628hc3a0,

    "clean_uncertainty_length":
        clean_uncertainty_risk_628hc3a0.shape
        == (n_dev_628hc3a0,),

    "clean_quality_length":
        clean_quality_risk_628hc3a0.shape
        == (n_dev_628hc3a0,),

    "prediction_matrix_finite":
        np.isfinite(prediction_matrix_628hc3a0).all(),

    "mc_std_matrix_finite":
        np.isfinite(mc_std_matrix_628hc3a0).all(),

    "quality_risk_matrix_finite":
        np.isfinite(quality_risk_matrix_628hc3a0).all(),

    "quality_risk_within_0_1":
        (
            np.nanmin(quality_risk_matrix_628hc3a0) >= 0.0
            and
            np.nanmax(quality_risk_matrix_628hc3a0) <= 1.0
        ),

    "clean_uncertainty_finite":
        np.isfinite(clean_uncertainty_risk_628hc3a0).all(),

    "clean_quality_finite":
        np.isfinite(clean_quality_risk_628hc3a0).all(),

    "clean_uncertainty_within_0_1":
        (
            np.nanmin(clean_uncertainty_risk_628hc3a0) >= 0.0
            and
            np.nanmax(clean_uncertainty_risk_628hc3a0) <= 1.0
        ),

    "clean_quality_within_0_1":
        (
            np.nanmin(clean_quality_risk_628hc3a0) >= 0.0
            and
            np.nanmax(clean_quality_risk_628hc3a0) <= 1.0
        ),
}

structural_table_628hc3a0 = pd.DataFrame(
    {
        "check": list(structural_checks_628hc3a0.keys()),
        "passed": list(structural_checks_628hc3a0.values()),
    }
)

print("\n--- STRUCTURAL / NUMERICAL AUDIT ---")
display(structural_table_628hc3a0)

if not structural_table_628hc3a0["passed"].all():
    failed_628hc3a0 = structural_table_628hc3a0.loc[
        ~structural_table_628hc3a0["passed"],
        "check"
    ].tolist()

    raise RuntimeError(
        "Step 6.28H-C3-A0 structural contract failed:\n"
        + "\n".join(f"  - {name}" for name in failed_628hc3a0)
    )


# ------------------------------------------------------------
# 4. MANIFEST ORDER CONTRACT
# ------------------------------------------------------------

required_manifest_cols_628hc3a0 = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
]

missing_manifest_cols_628hc3a0 = [
    col
    for col in required_manifest_cols_628hc3a0
    if col not in manifest_628hc3a0.columns
]

if missing_manifest_cols_628hc3a0:
    raise RuntimeError(
        "Manifest is missing required columns:\n"
        + "\n".join(
            f"  - {col}"
            for col in missing_manifest_cols_628hc3a0
        )
    )

expected_numbers_628hc3a0 = np.arange(
    1,
    n_scenarios_628hc3a0 + 1,
    dtype=int
)

actual_numbers_628hc3a0 = (
    manifest_628hc3a0["scenario_number"]
    .to_numpy(dtype=int)
)

expected_ids_628hc3a0 = np.array(
    [
        f"FD002_SCENARIO_{i:03d}"
        for i in expected_numbers_628hc3a0
    ],
    dtype=object
)

actual_ids_628hc3a0 = (
    manifest_628hc3a0["scenario_id"]
    .astype(str)
    .to_numpy()
)

scenario_numbers_exact_628hc3a0 = np.array_equal(
    actual_numbers_628hc3a0,
    expected_numbers_628hc3a0
)

scenario_ids_exact_628hc3a0 = np.array_equal(
    actual_ids_628hc3a0,
    expected_ids_628hc3a0
)

print("\n--- MANIFEST ORDER CONTRACT ---")
print(
    "Scenario numbers 001–315 exact :",
    scenario_numbers_exact_628hc3a0
)
print(
    "Scenario IDs exact              :",
    scenario_ids_exact_628hc3a0
)

if not (
    scenario_numbers_exact_628hc3a0
    and scenario_ids_exact_628hc3a0
):
    raise RuntimeError(
        "Frozen manifest ordering is not exact."
    )


# ------------------------------------------------------------
# 5. SEARCH NAMESPACE FOR EXISTING C1-D / C2-D OUTPUT OBJECTS
#
# Read-only diagnostic.
# We do NOT depend on these objects yet.
# ------------------------------------------------------------

namespace_snapshot_628hc3a0 = list(globals().items())

candidate_rows_628hc3a0 = []

keywords_628hc3a0 = (
    "628hc1d",
    "628hc2d",
    "uncertainty",
    "quality",
    "selective",
    "risk_matrix",
    "response",
)

for name_628hc3a0, obj_628hc3a0 in namespace_snapshot_628hc3a0:

    lname_628hc3a0 = name_628hc3a0.lower()

    if not any(
        key in lname_628hc3a0
        for key in keywords_628hc3a0
    ):
        continue

    obj_type_628hc3a0 = type(obj_628hc3a0).__name__

    try:
        shape_628hc3a0 = tuple(obj_628hc3a0.shape)
    except Exception:
        shape_628hc3a0 = None

    try:
        length_628hc3a0 = len(obj_628hc3a0)
    except Exception:
        length_628hc3a0 = None

    candidate_rows_628hc3a0.append(
        {
            "object_name": name_628hc3a0,
            "type": obj_type_628hc3a0,
            "shape": shape_628hc3a0,
            "length": length_628hc3a0,
        }
    )

candidate_table_628hc3a0 = (
    pd.DataFrame(candidate_rows_628hc3a0)
    .sort_values(
        by="object_name",
        key=lambda s: s.str.lower()
    )
    .reset_index(drop=True)
)

print("\n--- EXISTING COMPLEMENTARITY-RELEVANT OBJECTS ---")
print("Candidate objects found:", len(candidate_table_628hc3a0))

display(candidate_table_628hc3a0)


# ------------------------------------------------------------
# 6. CURRENT SIGNAL FINGERPRINTS
# ------------------------------------------------------------

def fingerprint_628hc3a0(values):
    arr = np.asarray(values, dtype=np.float64).reshape(-1)

    return {
        "n": int(arr.size),
        "min": float(np.min(arr)),
        "q25": float(np.quantile(arr, 0.25)),
        "median": float(np.median(arr)),
        "mean": float(np.mean(arr)),
        "q75": float(np.quantile(arr, 0.75)),
        "q95": float(np.quantile(arr, 0.95)),
        "max": float(np.max(arr)),
    }


signal_fingerprint_628hc3a0 = pd.DataFrame(
    [
        {
            "signal": "clean_uncertainty_risk",
            **fingerprint_628hc3a0(
                clean_uncertainty_risk_628hc3a0
            ),
        },
        {
            "signal": "clean_sensor_quality_risk",
            **fingerprint_628hc3a0(
                clean_quality_risk_628hc3a0
            ),
        },
        {
            "signal": "all_corrupted_mc_std",
            **fingerprint_628hc3a0(
                mc_std_matrix_628hc3a0
            ),
        },
        {
            "signal": "all_corrupted_sensor_quality_risk",
            **fingerprint_628hc3a0(
                quality_risk_matrix_628hc3a0
            ),
        },
    ]
)

print("\n--- CURRENT SIGNAL FINGERPRINTS ---")
display(signal_fingerprint_628hc3a0)


# ------------------------------------------------------------
# 7. CLEAN COMPLEMENTARITY DIAGNOSTIC
#
# This is descriptive only.
# No combination and no threshold selection.
# ------------------------------------------------------------

clean_signal_pearson_628hc3a0 = float(
    pd.Series(
        clean_uncertainty_risk_628hc3a0
    ).corr(
        pd.Series(
            clean_quality_risk_628hc3a0
        ),
        method="pearson",
    )
)

clean_signal_spearman_628hc3a0 = float(
    pd.Series(
        clean_uncertainty_risk_628hc3a0
    ).corr(
        pd.Series(
            clean_quality_risk_628hc3a0
        ),
        method="spearman",
    )
)

print("\n--- CLEAN SIGNAL ASSOCIATION ---")
print(
    "Uncertainty vs quality Pearson  :",
    clean_signal_pearson_628hc3a0
)
print(
    "Uncertainty vs quality Spearman :",
    clean_signal_spearman_628hc3a0
)

print(
    "\nInterpretation reminder: low/moderate association would "
    "support non-redundancy, but this diagnostic alone does "
    "NOT prove useful complementarity."
)


# ------------------------------------------------------------
# 8. FREEZE PRE-FLIGHT CONTRACT IN MEMORY
# ------------------------------------------------------------

FD002_COMPLEMENTARITY_PREFLIGHT_CONTRACT_628HC3A0 = {
    "dataset": "FD002",
    "population": "DEV only",
    "scenario_count": int(n_scenarios_628hc3a0),
    "dev_windows": int(n_dev_628hc3a0),
    "prediction_matrix_shape":
        tuple(prediction_matrix_628hc3a0.shape),
    "mc_std_matrix_shape":
        tuple(mc_std_matrix_628hc3a0.shape),
    "sensor_quality_matrix_shape":
        tuple(quality_risk_matrix_628hc3a0.shape),
    "manifest_order_exact":
        bool(
            scenario_numbers_exact_628hc3a0
            and scenario_ids_exact_628hc3a0
        ),
    "all_preflight_checks_passed":
        bool(structural_table_628hc3a0["passed"].all()),
    "clean_signal_pearson":
        clean_signal_pearson_628hc3a0,
    "clean_signal_spearman":
        clean_signal_spearman_628hc3a0,
    "official_test_labels_read": False,
    "test_scored": False,
    "combined_risk_constructed": False,
    "threshold_selected": False,
    "model_retrained": False,
    "files_written": False,
}


# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("STEP 6.28H-C3-A0 FINAL SUMMARY")
print("=" * 78)

print(
    "Frozen scenarios                 :",
    n_scenarios_628hc3a0
)
print(
    "DEV windows                      :",
    n_dev_628hc3a0
)
print(
    "Prediction matrix                :",
    prediction_matrix_628hc3a0.shape
)
print(
    "MC-STD matrix                    :",
    mc_std_matrix_628hc3a0.shape
)
print(
    "Sensor-quality matrix            :",
    quality_risk_matrix_628hc3a0.shape
)
print(
    "Manifest order exact             :",
    scenario_numbers_exact_628hc3a0
    and scenario_ids_exact_628hc3a0
)
print(
    "All pre-flight checks passed     :",
    structural_table_628hc3a0["passed"].all()
)
print(
    "Clean uncertainty-quality rho    :",
    clean_signal_spearman_628hc3a0
)

print("Official TEST labels read         : False")
print("TEST scored                       : False")
print("Combined risk constructed         : False")
print("Threshold selected                : False")
print("Model retrained                   : False")
print("Files written                     : False")

print("\n=== END STEP 6.28H-C3-A0 ===")

In [ ]:
print("=" * 70)
print("C2-B CLEAN SENSOR-QUALITY OBJECT INSPECTION")
print("=" * 70)

obj = CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B

print("Type:", type(obj).__name__)

if isinstance(obj, dict):
    print("\nDictionary keys:")
    for key, value in obj.items():
        try:
            shape = value.shape
        except Exception:
            shape = None

        try:
            length = len(value)
        except Exception:
            length = None

        print(
            f"{key:<40} "
            f"type={type(value).__name__:<15} "
            f"shape={str(shape):<20} "
            f"length={length}"
        )
else:
    print("Shape:", getattr(obj, "shape", None))
    print("Length:", len(obj) if hasattr(obj, "__len__") else None)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A0-R1
# COMBINED-RELIABILITY COMPLEMENTARITY PRE-FLIGHT
#
# PURPOSE
# -------
# 1. Recover the exact clean sensor-quality risk vector correctly.
# 2. Reconstruct the frozen uncertainty-risk chain for all 315
#    corrupted FD002 DEV scenarios.
# 3. Compare uncertainty risk and sensor-quality risk WITHOUT
#    selecting any combination rule or abstention threshold.
# 4. Preserve all previous frozen objects.
#
# IMPORTANT
# ---------
# - DEV only
# - No official TEST labels
# - No TEST scoring
# - No model retraining
# - No threshold selection
# - No files written
# ================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 78)
print("STEP 6.28H-C3-A0-R1 — COMBINED-RELIABILITY COMPLEMENTARITY PRE-FLIGHT")
print("=" * 78)


# ================================================================
# 1. REQUIRED OBJECTS
# ================================================================

required_objects_628hc3a0r1 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "Y_DEV_PRIMARY_FD002_628D",
    "MC_STD_MATRIX_FD002_628HC1C",

    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",

    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",

    "apply_phase_adjustment_628fbr1",
    "apply_g1_right_ecdf_628fc",
]

print("\n--- REQUIRED CORE OBJECTS ---")

missing_628hc3a0r1 = []

for name in required_objects_628hc3a0r1:
    present = name in globals()
    print(f"{name:<55} : {present}")
    if not present:
        missing_628hc3a0r1.append(name)

if missing_628hc3a0r1:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc3a0r1)
    )


# ================================================================
# 2. RECOVER FROZEN ARRAYS
# ================================================================

manifest_628hc3a0r1 = FD002_CORRUPTION_MANIFEST_628GD.copy()

prediction_matrix_628hc3a0r1 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

mc_std_matrix_628hc3a0r1 = np.asarray(
    MC_STD_MATRIX_FD002_628HC1C,
    dtype=np.float64
)

quality_risk_matrix_628hc3a0r1 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

y_dev_628hc3a0r1 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_uncertainty_risk_628hc3a0r1 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


# ------------------------------------------------
# CRITICAL CORRECTION
# ------------------------------------------------
clean_quality_container_628hc3a0r1 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

if not isinstance(clean_quality_container_628hc3a0r1, dict):
    raise TypeError(
        "Expected CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
        "to be a dictionary."
    )

if "sensor_quality_risk" not in clean_quality_container_628hc3a0r1:
    raise KeyError(
        "Key 'sensor_quality_risk' not found in "
        "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B."
    )

clean_quality_risk_628hc3a0r1 = np.asarray(
    clean_quality_container_628hc3a0r1["sensor_quality_risk"],
    dtype=np.float64
).reshape(-1)


# ================================================================
# 3. POPULATION CONTRACT
# ================================================================

n_scenarios_628hc3a0r1 = len(manifest_628hc3a0r1)
n_dev_628hc3a0r1 = len(y_dev_628hc3a0r1)

expected_matrix_shape_628hc3a0r1 = (
    n_scenarios_628hc3a0r1,
    n_dev_628hc3a0r1,
)

print("\n--- POPULATION CONTRACT ---")
print("Scenarios                 :", n_scenarios_628hc3a0r1)
print("DEV windows               :", n_dev_628hc3a0r1)
print("Prediction matrix         :", prediction_matrix_628hc3a0r1.shape)
print("MC-STD matrix             :", mc_std_matrix_628hc3a0r1.shape)
print("Sensor-quality matrix     :", quality_risk_matrix_628hc3a0r1.shape)
print("Clean uncertainty risk    :", clean_uncertainty_risk_628hc3a0r1.shape)
print("Clean sensor-quality risk :", clean_quality_risk_628hc3a0r1.shape)

if n_scenarios_628hc3a0r1 != 315:
    raise RuntimeError(
        f"Expected 315 scenarios; found {n_scenarios_628hc3a0r1}."
    )

if prediction_matrix_628hc3a0r1.shape != expected_matrix_shape_628hc3a0r1:
    raise RuntimeError(
        "Corrupted prediction matrix has unexpected shape."
    )

if mc_std_matrix_628hc3a0r1.shape != expected_matrix_shape_628hc3a0r1:
    raise RuntimeError(
        "MC-STD matrix has unexpected shape."
    )

if quality_risk_matrix_628hc3a0r1.shape != expected_matrix_shape_628hc3a0r1:
    raise RuntimeError(
        "Sensor-quality risk matrix has unexpected shape."
    )

if clean_uncertainty_risk_628hc3a0r1.shape != (n_dev_628hc3a0r1,):
    raise RuntimeError(
        "Clean uncertainty-risk vector has unexpected shape."
    )

if clean_quality_risk_628hc3a0r1.shape != (n_dev_628hc3a0r1,):
    raise RuntimeError(
        "Clean sensor-quality-risk vector has unexpected shape."
    )


# ================================================================
# 4. NUMERICAL INTEGRITY
# ================================================================

numeric_objects_628hc3a0r1 = {
    "prediction_matrix": prediction_matrix_628hc3a0r1,
    "mc_std_matrix": mc_std_matrix_628hc3a0r1,
    "quality_risk_matrix": quality_risk_matrix_628hc3a0r1,
    "y_dev": y_dev_628hc3a0r1,
    "clean_uncertainty_risk": clean_uncertainty_risk_628hc3a0r1,
    "clean_quality_risk": clean_quality_risk_628hc3a0r1,
}

print("\n--- NUMERICAL INTEGRITY ---")

for name, arr in numeric_objects_628hc3a0r1.items():
    finite = bool(np.isfinite(arr).all())
    print(f"{name:<28} finite={finite}")

    if not finite:
        raise RuntimeError(f"{name} contains non-finite values.")

for name, arr in {
    "quality_risk_matrix": quality_risk_matrix_628hc3a0r1,
    "clean_uncertainty_risk": clean_uncertainty_risk_628hc3a0r1,
    "clean_quality_risk": clean_quality_risk_628hc3a0r1,
}.items():

    within = bool(
        np.all(arr >= -1e-12)
        and np.all(arr <= 1.0 + 1e-12)
    )

    print(f"{name:<28} within_[0,1]={within}")

    if not within:
        raise RuntimeError(
            f"{name} contains values outside [0,1]."
        )


# ================================================================
# 5. WEIGHTED EMPIRICAL MID-RANK G2 TRANSFORM
# ================================================================

g2_ref_values_628hc3a0r1 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

g2_ref_weights_628hc3a0r1 = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

if len(g2_ref_values_628hc3a0r1) != len(g2_ref_weights_628hc3a0r1):
    raise RuntimeError(
        "G2 reference values and weights have unequal lengths."
    )

if not np.isfinite(g2_ref_values_628hc3a0r1).all():
    raise RuntimeError("G2 reference values are not fully finite.")

if not np.isfinite(g2_ref_weights_628hc3a0r1).all():
    raise RuntimeError("G2 reference weights are not fully finite.")

if np.any(g2_ref_weights_628hc3a0r1 < 0):
    raise RuntimeError("G2 reference weights contain negative values.")

weight_sum_628hc3a0r1 = float(
    g2_ref_weights_628hc3a0r1.sum()
)

if weight_sum_628hc3a0r1 <= 0:
    raise RuntimeError("G2 reference weight sum must be positive.")

g2_ref_weights_628hc3a0r1 = (
    g2_ref_weights_628hc3a0r1
    / weight_sum_628hc3a0r1
)

g2_order_628hc3a0r1 = np.argsort(
    g2_ref_values_628hc3a0r1,
    kind="mergesort"
)

g2_sorted_values_628hc3a0r1 = (
    g2_ref_values_628hc3a0r1[g2_order_628hc3a0r1]
)

g2_sorted_weights_628hc3a0r1 = (
    g2_ref_weights_628hc3a0r1[g2_order_628hc3a0r1]
)

g2_cumulative_weights_628hc3a0r1 = np.cumsum(
    g2_sorted_weights_628hc3a0r1
)


def weighted_midrank_ecdf_628hc3a0r1(values):
    """
    Weighted empirical mid-rank percentile:
        P(reference < x)
        + 0.5 * P(reference == x)

    Uses the frozen corrected FD002 G2 reference.
    """

    x = np.asarray(values, dtype=np.float64)

    left_idx = np.searchsorted(
        g2_sorted_values_628hc3a0r1,
        x,
        side="left",
    )

    right_idx = np.searchsorted(
        g2_sorted_values_628hc3a0r1,
        x,
        side="right",
    )

    below_mass = np.zeros_like(x, dtype=np.float64)
    through_mass = np.zeros_like(x, dtype=np.float64)

    left_mask = left_idx > 0
    right_mask = right_idx > 0

    below_mass[left_mask] = (
        g2_cumulative_weights_628hc3a0r1[
            left_idx[left_mask] - 1
        ]
    )

    through_mass[right_mask] = (
        g2_cumulative_weights_628hc3a0r1[
            right_idx[right_mask] - 1
        ]
    )

    equal_mass = through_mass - below_mass

    result = below_mass + 0.5 * equal_mass

    return np.clip(result, 0.0, 1.0)


print("\n--- G2 REFERENCE CONTRACT ---")
print("Reference values :", len(g2_ref_values_628hc3a0r1))
print("Reference weights:", len(g2_ref_weights_628hc3a0r1))
print(
    "Weight sum      :",
    float(g2_ref_weights_628hc3a0r1.sum())
)


# ================================================================
# 6. CLEAN RELIABILITY-CHAIN REPRODUCTION
# ================================================================

clean_adjusted_rebuilt_628hc3a0r1 = np.asarray(
    apply_phase_adjustment_628fbr1(
        DEV_PREDICTIONS_FD002_628EB,
        DEV_MC_STD_FD002_628FAR2,
    ),
    dtype=np.float64,
).reshape(-1)

clean_g1_rebuilt_628hc3a0r1 = np.asarray(
    apply_g1_right_ecdf_628fc(
        clean_adjusted_rebuilt_628hc3a0r1
    ),
    dtype=np.float64,
).reshape(-1)

clean_final_rebuilt_628hc3a0r1 = (
    weighted_midrank_ecdf_628hc3a0r1(
        clean_g1_rebuilt_628hc3a0r1
    )
)

clean_adjusted_error_628hc3a0r1 = float(
    np.max(
        np.abs(
            clean_adjusted_rebuilt_628hc3a0r1
            - np.asarray(
                FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1,
                dtype=np.float64,
            ).reshape(-1)
        )
    )
)

clean_g1_error_628hc3a0r1 = float(
    np.max(
        np.abs(
            clean_g1_rebuilt_628hc3a0r1
            - np.asarray(
                FD002_DEV_G1_UNCERTAINTY_RISK_628FC,
                dtype=np.float64,
            ).reshape(-1)
        )
    )
)

clean_final_error_628hc3a0r1 = float(
    np.max(
        np.abs(
            clean_final_rebuilt_628hc3a0r1
            - clean_uncertainty_risk_628hc3a0r1
        )
    )
)

print("\n--- CLEAN RELIABILITY-CHAIN REPRODUCTION ---")
print(
    "Phase-adjustment maximum error :",
    clean_adjusted_error_628hc3a0r1
)
print(
    "G1 maximum error               :",
    clean_g1_error_628hc3a0r1
)
print(
    "Final-risk maximum error       :",
    clean_final_error_628hc3a0r1
)

clean_chain_exact_628hc3a0r1 = bool(
    clean_adjusted_error_628hc3a0r1 <= 1e-12
    and clean_g1_error_628hc3a0r1 <= 1e-12
    and clean_final_error_628hc3a0r1 <= 1e-12
)

print(
    "Complete clean chain exact     :",
    clean_chain_exact_628hc3a0r1
)

if not clean_chain_exact_628hc3a0r1:
    raise RuntimeError(
        "Frozen clean uncertainty chain was not reproduced exactly."
    )


# ================================================================
# 7. RECONSTRUCT UNCERTAINTY RISK FOR ALL CORRUPTED SCENARIOS
# ================================================================

uncertainty_risk_matrix_628hc3a0r1 = np.empty(
    expected_matrix_shape_628hc3a0r1,
    dtype=np.float64,
)

print(
    "\n--- RECONSTRUCTING FROZEN UNCERTAINTY RISK "
    "FOR 315 CORRUPTED SCENARIOS ---"
)

for i in range(n_scenarios_628hc3a0r1):

    corrupted_pred_i = prediction_matrix_628hc3a0r1[i]
    corrupted_mcstd_i = mc_std_matrix_628hc3a0r1[i]

    adjusted_i = np.asarray(
        apply_phase_adjustment_628fbr1(
            corrupted_pred_i,
            corrupted_mcstd_i,
        ),
        dtype=np.float64,
    ).reshape(-1)

    g1_i = np.asarray(
        apply_g1_right_ecdf_628fc(adjusted_i),
        dtype=np.float64,
    ).reshape(-1)

    final_i = weighted_midrank_ecdf_628hc3a0r1(g1_i)

    uncertainty_risk_matrix_628hc3a0r1[i] = final_i

    if (
        i == 0
        or (i + 1) % 50 == 0
        or (i + 1) == n_scenarios_628hc3a0r1
    ):
        row = manifest_628hc3a0r1.iloc[i]

        print(
            f"[{i+1:03d}/{n_scenarios_628hc3a0r1}] "
            f"{row['scenario_id']} | "
            f"{row['family']} | "
            f"{row['severity']} | "
            f"unc_med={np.median(final_i):.4f} | "
            f"quality_med="
            f"{np.median(quality_risk_matrix_628hc3a0r1[i]):.4f}"
        )


if not np.isfinite(
    uncertainty_risk_matrix_628hc3a0r1
).all():
    raise RuntimeError(
        "Reconstructed corrupted uncertainty-risk matrix "
        "contains non-finite values."
    )

if not (
    np.all(
        uncertainty_risk_matrix_628hc3a0r1 >= -1e-12
    )
    and np.all(
        uncertainty_risk_matrix_628hc3a0r1 <= 1.0 + 1e-12
    )
):
    raise RuntimeError(
        "Reconstructed uncertainty risk contains values "
        "outside [0,1]."
    )

print("\nReconstructed uncertainty matrix:",
      uncertainty_risk_matrix_628hc3a0r1.shape)


# ================================================================
# 8. SAFE SPEARMAN HELPER
# ================================================================

def safe_spearman_628hc3a0r1(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    b = np.asarray(b, dtype=np.float64).reshape(-1)

    if len(a) != len(b):
        raise ValueError("Spearman arrays have unequal lengths.")

    if np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan

    result = spearmanr(a, b)

    if hasattr(result, "statistic"):
        return float(result.statistic)

    return float(result[0])


# ================================================================
# 9. CLEAN COMPLEMENTARITY DIAGNOSTIC
# ================================================================

clean_abs_error_628hc3a0r1 = np.abs(
    np.asarray(
        DEV_PREDICTIONS_FD002_628EB,
        dtype=np.float64
    ).reshape(-1)
    - y_dev_628hc3a0r1
)

clean_unc_vs_quality_628hc3a0r1 = (
    safe_spearman_628hc3a0r1(
        clean_uncertainty_risk_628hc3a0r1,
        clean_quality_risk_628hc3a0r1,
    )
)

clean_unc_vs_error_628hc3a0r1 = (
    safe_spearman_628hc3a0r1(
        clean_uncertainty_risk_628hc3a0r1,
        clean_abs_error_628hc3a0r1,
    )
)

clean_quality_vs_error_628hc3a0r1 = (
    safe_spearman_628hc3a0r1(
        clean_quality_risk_628hc3a0r1,
        clean_abs_error_628hc3a0r1,
    )
)

print("\n--- CLEAN COMPLEMENTARITY DIAGNOSTIC ---")
print(
    "Uncertainty-risk vs quality-risk Spearman :",
    clean_unc_vs_quality_628hc3a0r1
)
print(
    "Uncertainty-risk vs absolute-error        :",
    clean_unc_vs_error_628hc3a0r1
)
print(
    "Quality-risk vs absolute-error            :",
    clean_quality_vs_error_628hc3a0r1
)


# ================================================================
# 10. CORRUPTED-SCENARIO COMPLEMENTARITY
# ================================================================

complementarity_rows_628hc3a0r1 = []

for i in range(n_scenarios_628hc3a0r1):

    row = manifest_628hc3a0r1.iloc[i]

    pred_i = prediction_matrix_628hc3a0r1[i]
    unc_i = uncertainty_risk_matrix_628hc3a0r1[i]
    quality_i = quality_risk_matrix_628hc3a0r1[i]

    abs_error_i = np.abs(
        pred_i - y_dev_628hc3a0r1
    )

    complementarity_rows_628hc3a0r1.append({
        "scenario_id": row["scenario_id"],
        "scenario_number": int(row["scenario_number"]),
        "family": row["family"],
        "sensor": row["sensor"],
        "severity": row["severity"],

        "full_mae": float(np.mean(abs_error_i)),

        "uncertainty_risk_median":
            float(np.median(unc_i)),

        "sensor_quality_risk_median":
            float(np.median(quality_i)),

        "uncertainty_quality_spearman":
            safe_spearman_628hc3a0r1(
                unc_i,
                quality_i,
            ),

        "uncertainty_error_spearman":
            safe_spearman_628hc3a0r1(
                unc_i,
                abs_error_i,
            ),

        "quality_error_spearman":
            safe_spearman_628hc3a0r1(
                quality_i,
                abs_error_i,
            ),

        "quality_risk_unique":
            int(np.unique(quality_i).size),

        "uncertainty_risk_unique":
            int(np.unique(unc_i).size),
    })


FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1 = pd.DataFrame(
    complementarity_rows_628hc3a0r1
)


# ================================================================
# 11. FAMILY × SEVERITY SUMMARY
# ================================================================

FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1
    .groupby(
        ["family", "severity"],
        observed=True
    )
    .agg(
        scenarios=("scenario_id", "size"),

        median_full_mae=(
            "full_mae",
            "median"
        ),

        median_uncertainty_quality_spearman=(
            "uncertainty_quality_spearman",
            "median"
        ),

        median_uncertainty_error_spearman=(
            "uncertainty_error_spearman",
            "median"
        ),

        median_quality_error_spearman=(
            "quality_error_spearman",
            "median"
        ),

        median_uncertainty_risk=(
            "uncertainty_risk_median",
            "median"
        ),

        median_quality_risk=(
            "sensor_quality_risk_median",
            "median"
        ),
    )
    .reset_index()
)

severity_order_628hc3a0r1 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_order_628hc3a0r1 = {
    "bias": 0,
    "noise": 1,
    "drift": 2,
    "dropout": 3,
    "stuck": 4,
}

FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1[
    "_family_order"
] = (
    FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1[
        "family"
    ]
    .map(family_order_628hc3a0r1)
)

FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1[
    "_severity_order"
] = (
    FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1[
        "severity"
    ]
    .map(severity_order_628hc3a0r1)
)

FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1 = (
    FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1
    .sort_values(
        ["_family_order", "_severity_order"]
    )
    .drop(
        columns=[
            "_family_order",
            "_severity_order",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n--- FAMILY × SEVERITY COMPLEMENTARITY RESPONSE ---"
)

display(
    FD002_COMPLEMENTARITY_FAMILY_SUMMARY_628HC3A0R1
)


# ================================================================
# 12. OVERALL COMPLEMENTARITY FINGERPRINT
# ================================================================

finite_unc_quality_628hc3a0r1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1[
        "uncertainty_quality_spearman"
    ].dropna()
)

finite_unc_error_628hc3a0r1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1[
        "uncertainty_error_spearman"
    ].dropna()
)

finite_quality_error_628hc3a0r1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1[
        "quality_error_spearman"
    ].dropna()
)

print("\n--- OVERALL COMPLEMENTARITY FINGERPRINT ---")

print(
    "Finite uncertainty-quality correlations :",
    f"{len(finite_unc_quality_628hc3a0r1)} / "
    f"{n_scenarios_628hc3a0r1}"
)

print(
    "Median uncertainty-quality Spearman      :",
    float(
        finite_unc_quality_628hc3a0r1.median()
    )
    if len(finite_unc_quality_628hc3a0r1)
    else np.nan
)

print(
    "Median uncertainty-error Spearman        :",
    float(
        finite_unc_error_628hc3a0r1.median()
    )
    if len(finite_unc_error_628hc3a0r1)
    else np.nan
)

print(
    "Median quality-error Spearman            :",
    float(
        finite_quality_error_628hc3a0r1.median()
    )
    if len(finite_quality_error_628hc3a0r1)
    else np.nan
)


# ================================================================
# 13. STUCK VS NON-STUCK DIAGNOSTIC
# ================================================================

stuck_mask_628hc3a0r1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1[
        "family"
    ] == "stuck"
)

stuck_summary_628hc3a0r1 = (
    FD002_COMPLEMENTARITY_RESPONSE_628HC3A0R1
    .assign(
        population=np.where(
            stuck_mask_628hc3a0r1,
            "stuck",
            "non_stuck"
        )
    )
    .groupby(
        "population",
        observed=True
    )
    .agg(
        scenarios=("scenario_id", "size"),

        median_full_mae=(
            "full_mae",
            "median"
        ),

        median_uncertainty_error_spearman=(
            "uncertainty_error_spearman",
            "median"
        ),

        median_quality_error_spearman=(
            "quality_error_spearman",
            "median"
        ),

        median_uncertainty_quality_spearman=(
            "uncertainty_quality_spearman",
            "median"
        ),

        median_quality_risk=(
            "sensor_quality_risk_median",
            "median"
        ),
    )
    .reset_index()
)

print("\n--- STUCK VS NON-STUCK COMPLEMENTARITY ---")
display(stuck_summary_628hc3a0r1)


# ================================================================
# 14. FINAL INTEGRITY AUDIT
# ================================================================

audit_rows_628hc3a0r1 = [
    {
        "check": "scenario_count_315",
        "passed":
            n_scenarios_628hc3a0r1 == 315,
    },
    {
        "check": "prediction_matrix_shape",
        "passed":
            prediction_matrix_628hc3a0r1.shape
            == (315, 8367),
    },
    {
        "check": "mc_std_matrix_shape",
        "passed":
            mc_std_matrix_628hc3a0r1.shape
            == (315, 8367),
    },
    {
        "check": "quality_matrix_shape",
        "passed":
            quality_risk_matrix_628hc3a0r1.shape
            == (315, 8367),
    },
    {
        "check": "uncertainty_matrix_shape",
        "passed":
            uncertainty_risk_matrix_628hc3a0r1.shape
            == (315, 8367),
    },
    {
        "check": "uncertainty_matrix_finite",
        "passed":
            bool(
                np.isfinite(
                    uncertainty_risk_matrix_628hc3a0r1
                ).all()
            ),
    },
    {
        "check": "quality_matrix_finite",
        "passed":
            bool(
                np.isfinite(
                    quality_risk_matrix_628hc3a0r1
                ).all()
            ),
    },
    {
        "check": "clean_chain_exact",
        "passed":
            clean_chain_exact_628hc3a0r1,
    },
    {
        "check": "clean_quality_vector_recovered",
        "passed":
            clean_quality_risk_628hc3a0r1.shape
            == (8367,),
    },
]

FD002_COMPLEMENTARITY_AUDIT_628HC3A0R1 = pd.DataFrame(
    audit_rows_628hc3a0r1
)

print("\n--- FINAL PRE-FLIGHT AUDIT ---")
display(FD002_COMPLEMENTARITY_AUDIT_628HC3A0R1)

all_checks_passed_628hc3a0r1 = bool(
    FD002_COMPLEMENTARITY_AUDIT_628HC3A0R1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a0r1:
    raise RuntimeError(
        "Step 6.28H-C3-A0-R1 integrity audit failed."
    )


# ================================================================
# 15. FREEZE IN-MEMORY CONTRACT
# ================================================================

FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1 = (
    uncertainty_risk_matrix_628hc3a0r1.copy()
)

FD002_CLEAN_QUALITY_RISK_VECTOR_628HC3A0R1 = (
    clean_quality_risk_628hc3a0r1.copy()
)

FD002_COMPLEMENTARITY_CONTRACT_628HC3A0R1 = {
    "scenarios": n_scenarios_628hc3a0r1,
    "dev_windows": n_dev_628hc3a0r1,

    "clean_quality_source":
        "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B"
        "['sensor_quality_risk']",

    "uncertainty_source":
        "frozen phase-adjustment -> G1 -> corrected G2",

    "quality_source":
        "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",

    "combination_rule_selected": False,
    "combination_weight_tuned": False,
    "abstention_threshold_selected": False,

    "dev_rul_labels_used_for_diagnostic": True,
    "official_test_labels_read": False,
    "test_scored": False,
    "model_retrained": False,
    "files_written": False,

    "all_checks_passed":
        all_checks_passed_628hc3a0r1,
}


# ================================================================
# 16. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 78)
print("STEP 6.28H-C3-A0-R1 FINAL SUMMARY")
print("=" * 78)

print(
    "Scenarios evaluated                 :",
    n_scenarios_628hc3a0r1
)

print(
    "DEV windows/scenario               :",
    n_dev_628hc3a0r1
)

print(
    "Uncertainty-risk matrix            :",
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1.shape
)

print(
    "Sensor-quality-risk matrix         :",
    quality_risk_matrix_628hc3a0r1.shape
)

print(
    "Clean quality vector source        :",
    "dictionary['sensor_quality_risk']"
)

print(
    "Complete clean uncertainty chain   :",
    clean_chain_exact_628hc3a0r1
)

print(
    "All pre-flight checks passed       :",
    all_checks_passed_628hc3a0r1
)

print(
    "Combination rule selected          : False"
)

print(
    "Combination weight tuned           : False"
)

print(
    "Abstention threshold selected      : False"
)

print(
    "Official TEST labels read          : False"
)

print(
    "TEST scored                        : False"
)

print(
    "Model retrained                    : False"
)

print(
    "Files written                      : False"
)

print("\n=== END STEP 6.28H-C3-A0-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A1
# CANDIDATE COMBINED-RELIABILITY FUSION EVALUATION
#
# PURPOSE
# -------
# Compare several label-free reliability fusion rules using DEV only.
#
# IMPORTANT
# ---------
# - No TEST labels are read.
# - No TEST scoring is performed.
# - No abstention threshold is selected.
# - No model is retrained.
# - No final combination rule is frozen here.
# - Existing uncertainty and sensor-quality branches remain unchanged.
# ================================================================

import time
import numpy as np
import pandas as pd


print("=" * 82)
print("STEP 6.28H-C3-A1 — CANDIDATE COMBINED-RELIABILITY FUSION EVALUATION")
print("=" * 82)


# ================================================================
# 1. REQUIRED OBJECTS
# ================================================================

required_objects_628hc3a1 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "Y_DEV_PRIMARY_FD002_628D",

    "MC_STD_MATRIX_FD002_628HC1C",

    "FD002_DEV_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_DEV_G1_UNCERTAINTY_RISK_628FC",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",

    "DEV_PREDICTIONS_FD002_628EB",
    "DEV_MC_STD_FD002_628FAR2",

    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",

    "apply_phase_adjustment_628fbr1",
    "apply_g1_right_ecdf_628fc",

    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc3a1 = []

for name in required_objects_628hc3a1:
    ok = name in globals()
    print(f"{name:<52} : {ok}")
    if not ok:
        missing_628hc3a1.append(name)

if missing_628hc3a1:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing_628hc3a1)
    )


# ================================================================
# 2. LOCAL COPIES / SHAPE CONTRACT
# ================================================================

manifest_628hc3a1 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

pred_matrix_628hc3a1 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

mc_std_matrix_628hc3a1 = np.asarray(
    MC_STD_MATRIX_FD002_628HC1C,
    dtype=np.float64
)

quality_matrix_628hc3a1 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

y_dev_628hc3a1 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_pred_628hc3a1 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64
).reshape(-1)

clean_mc_std_628hc3a1 = np.asarray(
    DEV_MC_STD_FD002_628FAR2,
    dtype=np.float64
).reshape(-1)

clean_uncertainty_stored_628hc3a1 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


# Clean sensor-quality object is intentionally a dictionary.
clean_quality_object_628hc3a1 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

if isinstance(clean_quality_object_628hc3a1, dict):

    if "sensor_quality_risk" not in clean_quality_object_628hc3a1:
        raise RuntimeError(
            "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
            "does not contain key 'sensor_quality_risk'."
        )

    clean_quality_628hc3a1 = np.asarray(
        clean_quality_object_628hc3a1["sensor_quality_risk"],
        dtype=np.float64
    ).reshape(-1)

else:
    clean_quality_628hc3a1 = np.asarray(
        clean_quality_object_628hc3a1,
        dtype=np.float64
    ).reshape(-1)


n_scenarios_628hc3a1 = len(manifest_628hc3a1)
n_dev_628hc3a1 = len(y_dev_628hc3a1)

expected_shape_628hc3a1 = (
    n_scenarios_628hc3a1,
    n_dev_628hc3a1
)


print("\n--- POPULATION CONTRACT ---")
print("Scenarios                    :", n_scenarios_628hc3a1)
print("DEV windows                  :", n_dev_628hc3a1)
print("Prediction matrix            :", pred_matrix_628hc3a1.shape)
print("MC-STD matrix                :", mc_std_matrix_628hc3a1.shape)
print("Sensor-quality matrix        :", quality_matrix_628hc3a1.shape)
print("Clean uncertainty risk       :", clean_uncertainty_stored_628hc3a1.shape)
print("Clean sensor-quality risk    :", clean_quality_628hc3a1.shape)


if n_scenarios_628hc3a1 != 315:
    raise RuntimeError(
        f"Expected 315 scenarios, got {n_scenarios_628hc3a1}."
    )

if pred_matrix_628hc3a1.shape != expected_shape_628hc3a1:
    raise RuntimeError("Prediction-matrix shape mismatch.")

if mc_std_matrix_628hc3a1.shape != expected_shape_628hc3a1:
    raise RuntimeError("MC-STD-matrix shape mismatch.")

if quality_matrix_628hc3a1.shape != expected_shape_628hc3a1:
    raise RuntimeError("Sensor-quality-matrix shape mismatch.")

if len(clean_pred_628hc3a1) != n_dev_628hc3a1:
    raise RuntimeError("Clean prediction length mismatch.")

if len(clean_mc_std_628hc3a1) != n_dev_628hc3a1:
    raise RuntimeError("Clean MC-STD length mismatch.")

if len(clean_uncertainty_stored_628hc3a1) != n_dev_628hc3a1:
    raise RuntimeError("Clean uncertainty-risk length mismatch.")

if len(clean_quality_628hc3a1) != n_dev_628hc3a1:
    raise RuntimeError("Clean sensor-quality-risk length mismatch.")


# ================================================================
# 3. NUMERICAL INTEGRITY
# ================================================================

integrity_items_628hc3a1 = {
    "prediction_matrix":
        np.isfinite(pred_matrix_628hc3a1).all(),

    "mc_std_matrix":
        np.isfinite(mc_std_matrix_628hc3a1).all(),

    "quality_matrix":
        np.isfinite(quality_matrix_628hc3a1).all(),

    "y_dev":
        np.isfinite(y_dev_628hc3a1).all(),

    "clean_prediction":
        np.isfinite(clean_pred_628hc3a1).all(),

    "clean_mc_std":
        np.isfinite(clean_mc_std_628hc3a1).all(),

    "clean_uncertainty":
        np.isfinite(clean_uncertainty_stored_628hc3a1).all(),

    "clean_quality":
        np.isfinite(clean_quality_628hc3a1).all(),

    "quality_matrix_within_0_1":
        bool(
            np.min(quality_matrix_628hc3a1) >= -1e-12
            and
            np.max(quality_matrix_628hc3a1) <= 1.0 + 1e-12
        ),

    "clean_quality_within_0_1":
        bool(
            np.min(clean_quality_628hc3a1) >= -1e-12
            and
            np.max(clean_quality_628hc3a1) <= 1.0 + 1e-12
        ),
}

print("\n--- NUMERICAL INTEGRITY ---")

for name, value in integrity_items_628hc3a1.items():
    print(f"{name:<32} : {value}")

if not all(integrity_items_628hc3a1.values()):
    raise RuntimeError(
        "Numerical-integrity check failed."
    )


# ================================================================
# 4. RECONSTRUCT EXACT G2 TRANSFORM
# ================================================================

g2_reference_values_628hc3a1 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

g2_reference_weights_628hc3a1 = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64
).reshape(-1)

if (
    len(g2_reference_values_628hc3a1)
    != len(g2_reference_weights_628hc3a1)
):
    raise RuntimeError(
        "G2 reference-value / weight length mismatch."
    )

if not np.isfinite(g2_reference_values_628hc3a1).all():
    raise RuntimeError("G2 reference values are non-finite.")

if not np.isfinite(g2_reference_weights_628hc3a1).all():
    raise RuntimeError("G2 reference weights are non-finite.")

if np.any(g2_reference_weights_628hc3a1 < 0):
    raise RuntimeError("Negative G2 reference weight detected.")


g2_order_628hc3a1 = np.argsort(
    g2_reference_values_628hc3a1,
    kind="mergesort"
)

g2_values_sorted_628hc3a1 = (
    g2_reference_values_628hc3a1[
        g2_order_628hc3a1
    ]
)

g2_weights_sorted_628hc3a1 = (
    g2_reference_weights_628hc3a1[
        g2_order_628hc3a1
    ]
)

g2_cumulative_weights_628hc3a1 = np.cumsum(
    g2_weights_sorted_628hc3a1
)


def apply_g2_right_weighted_ecdf_628hc3a1(values):
    """
    Weighted right empirical CDF using the frozen G2 reference.
    """

    x = np.asarray(
        values,
        dtype=np.float64
    )

    idx = np.searchsorted(
        g2_values_sorted_628hc3a1,
        x,
        side="right"
    ) - 1

    result = np.zeros_like(
        x,
        dtype=np.float64
    )

    valid = idx >= 0

    result[valid] = (
        g2_cumulative_weights_628hc3a1[
            idx[valid]
        ]
    )

    return np.clip(
        result,
        0.0,
        1.0
    )


print("\n--- G2 REFERENCE CONTRACT ---")
print(
    "Reference values  :",
    len(g2_reference_values_628hc3a1)
)
print(
    "Reference weights :",
    len(g2_reference_weights_628hc3a1)
)
print(
    "Weight sum        :",
    float(np.sum(g2_reference_weights_628hc3a1))
)


# ================================================================
# 5. EXACT CLEAN UNCERTAINTY-CHAIN REPRODUCTION
# ================================================================

clean_adjusted_rebuilt_628hc3a1 = np.asarray(
    apply_phase_adjustment_628fbr1(
        clean_pred_628hc3a1,
        clean_mc_std_628hc3a1
    ),
    dtype=np.float64
).reshape(-1)

clean_g1_rebuilt_628hc3a1 = np.asarray(
    apply_g1_right_ecdf_628fc(
        clean_adjusted_rebuilt_628hc3a1
    ),
    dtype=np.float64
).reshape(-1)

clean_uncertainty_rebuilt_628hc3a1 = (
    apply_g2_right_weighted_ecdf_628hc3a1(
        clean_g1_rebuilt_628hc3a1
    )
)

clean_uncertainty_max_error_628hc3a1 = float(
    np.max(
        np.abs(
            clean_uncertainty_rebuilt_628hc3a1
            -
            clean_uncertainty_stored_628hc3a1
        )
    )
)

clean_uncertainty_exact_628hc3a1 = bool(
    clean_uncertainty_max_error_628hc3a1
    <= 1e-12
)

print("\n--- CLEAN UNCERTAINTY-CHAIN REPRODUCTION ---")
print(
    "Maximum absolute error :",
    clean_uncertainty_max_error_628hc3a1
)
print(
    "Exact <= 1e-12         :",
    clean_uncertainty_exact_628hc3a1
)

if not clean_uncertainty_exact_628hc3a1:
    raise RuntimeError(
        "Frozen clean uncertainty chain was not "
        "reproduced exactly."
    )


# ================================================================
# 6. RECONSTRUCT UNCERTAINTY RISK FOR 315 CORRUPTED SCENARIOS
# ================================================================

print(
    "\n--- RECONSTRUCTING CORRUPTED UNCERTAINTY-RISK MATRIX ---"
)

t0_unc_628hc3a1 = time.perf_counter()

uncertainty_matrix_628hc3a1 = np.empty(
    expected_shape_628hc3a1,
    dtype=np.float64
)

for i in range(n_scenarios_628hc3a1):

    adjusted_i = np.asarray(
        apply_phase_adjustment_628fbr1(
            pred_matrix_628hc3a1[i],
            mc_std_matrix_628hc3a1[i]
        ),
        dtype=np.float64
    ).reshape(-1)

    g1_i = np.asarray(
        apply_g1_right_ecdf_628fc(
            adjusted_i
        ),
        dtype=np.float64
    ).reshape(-1)

    uncertainty_matrix_628hc3a1[i] = (
        apply_g2_right_weighted_ecdf_628hc3a1(
            g1_i
        )
    )

    if (
        i == 0
        or (i + 1) % 50 == 0
        or i == n_scenarios_628hc3a1 - 1
    ):
        row = manifest_628hc3a1.iloc[i]

        print(
            f"[{i+1:03d}/{n_scenarios_628hc3a1}] "
            f"{row['scenario_id']} | "
            f"{row['family']} | "
            f"{row['severity']} | "
            f"Umedian="
            f"{np.median(uncertainty_matrix_628hc3a1[i]):.4f}"
        )


unc_runtime_628hc3a1 = (
    time.perf_counter()
    -
    t0_unc_628hc3a1
)

if not np.isfinite(
    uncertainty_matrix_628hc3a1
).all():
    raise RuntimeError(
        "Reconstructed uncertainty matrix contains "
        "non-finite values."
    )

if (
    np.min(uncertainty_matrix_628hc3a1) < -1e-12
    or
    np.max(uncertainty_matrix_628hc3a1) > 1.0 + 1e-12
):
    raise RuntimeError(
        "Reconstructed uncertainty risk falls "
        "outside [0,1]."
    )


print(
    "\nReconstructed uncertainty matrix:",
    uncertainty_matrix_628hc3a1.shape
)
print(
    "Runtime (sec):",
    unc_runtime_628hc3a1
)


# Freeze a working in-memory reconstruction for subsequent C3 steps.
FD002_UNCERTAINTY_RISK_MATRIX_REBUILT_628HC3A1 = (
    uncertainty_matrix_628hc3a1
)


# ================================================================
# 7. FIT-CALIBRATED SENSOR-QUALITY EXCESS
# ================================================================

fit_q95_628hc3a1 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)

if not np.isfinite(fit_q95_628hc3a1):
    raise RuntimeError(
        "FIT sensor-quality q95 marker is non-finite."
    )

if not (0.0 < fit_q95_628hc3a1 < 1.0):
    raise RuntimeError(
        "FIT sensor-quality q95 marker must lie "
        "strictly between 0 and 1."
    )


def quality_excess_628hc3a1(quality_risk):
    """
    FIT-q95-gated anomaly excess.

    0  -> quality risk at/below FIT q95
    1  -> quality risk reaches 1
    """

    q = np.asarray(
        quality_risk,
        dtype=np.float64
    )

    excess = (
        q - fit_q95_628hc3a1
    ) / (
        1.0 - fit_q95_628hc3a1
    )

    return np.clip(
        excess,
        0.0,
        1.0
    )


print("\n--- FIT-CALIBRATED QUALITY GATE ---")
print(
    "FIT quality-risk q95 marker :",
    fit_q95_628hc3a1
)
print(
    "Clean DEV fraction > q95    :",
    float(
        np.mean(
            clean_quality_628hc3a1
            >
            fit_q95_628hc3a1
        )
    )
)


# ================================================================
# 8. CANDIDATE FUSION RULES
# ================================================================

candidate_names_628hc3a1 = [
    "uncertainty_only",
    "sensor_quality_only",
    "simple_mean",
    "maximum",
    "fit_q95_gated_or",
    "fit_q95_normalized_add",
]


def build_candidate_risks_628hc3a1(
    uncertainty_risk,
    quality_risk
):
    """
    Return all candidate risk vectors.

    Lower score = accepted earlier.

    No RUL labels are used inside this function.
    """

    u = np.asarray(
        uncertainty_risk,
        dtype=np.float64
    )

    q = np.asarray(
        quality_risk,
        dtype=np.float64
    )

    if u.shape != q.shape:
        raise RuntimeError(
            "Uncertainty / quality vector shape mismatch."
        )

    e = quality_excess_628hc3a1(q)

    # 1. Frozen uncertainty baseline.
    uncertainty_only = u

    # 2. Frozen sensor-quality baseline.
    sensor_quality_only = q

    # 3. Simple equal-weight control.
    simple_mean = 0.5 * (u + q)

    # 4. Conservative max control.
    maximum = np.maximum(u, q)

    # 5. FIT-q95-gated probabilistic OR.
    #
    # If e = 0:
    #     result = u
    #
    # If e rises:
    #     sensor anomaly increases risk.
    #
    fit_q95_gated_or = (
        1.0
        -
        (1.0 - u)
        *
        (1.0 - e)
    )

    # 6. FIT-q95-gated normalized additive fusion.
    #
    # Properties:
    # - e = 0 -> exactly u
    # - bounded in [0,1]
    # - monotonic in both u and e
    # - if e is constant at 1, uncertainty ranking is preserved
    #
    fit_q95_normalized_add = (
        u + e
    ) / (
        1.0 + e
    )

    result = {
        "uncertainty_only":
            uncertainty_only,

        "sensor_quality_only":
            sensor_quality_only,

        "simple_mean":
            simple_mean,

        "maximum":
            maximum,

        "fit_q95_gated_or":
            fit_q95_gated_or,

        "fit_q95_normalized_add":
            fit_q95_normalized_add,
    }

    for name, values in result.items():

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"Candidate '{name}' contains "
                "non-finite values."
            )

        if (
            np.min(values) < -1e-12
            or
            np.max(values) > 1.0 + 1e-12
        ):
            raise RuntimeError(
                f"Candidate '{name}' falls "
                "outside [0,1]."
            )

    return result


# ================================================================
# 9. SAFE SPEARMAN
# ================================================================

def safe_spearman_628hc3a1(x, y):

    x = np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)

    y = np.asarray(
        y,
        dtype=np.float64
    ).reshape(-1)

    if len(x) != len(y):
        raise RuntimeError(
            "Spearman input length mismatch."
        )

    if len(x) < 2:
        return np.nan

    if np.all(x == x[0]):
        return np.nan

    if np.all(y == y[0]):
        return np.nan

    rx = pd.Series(x).rank(
        method="average"
    ).to_numpy(dtype=np.float64)

    ry = pd.Series(y).rank(
        method="average"
    ).to_numpy(dtype=np.float64)

    corr = np.corrcoef(
        rx,
        ry
    )[0, 1]

    return float(corr)


# ================================================================
# 10. TIE-AWARE SELECTIVE MAE
# ================================================================

COVERAGES_628HC3A1 = (
    0.90,
    0.80,
    0.70
)


def tie_aware_selective_mae_628hc3a1(
    abs_error,
    risk,
    coverages=COVERAGES_628HC3A1
):
    """
    Lower risk is accepted first.

    At the coverage boundary, if multiple samples have
    exactly the same risk, use expected error within that
    tied group rather than arbitrary row ordering.
    """

    err = np.asarray(
        abs_error,
        dtype=np.float64
    ).reshape(-1)

    risk = np.asarray(
        risk,
        dtype=np.float64
    ).reshape(-1)

    if len(err) != len(risk):
        raise RuntimeError(
            "Error/risk length mismatch."
        )

    if not np.isfinite(err).all():
        raise RuntimeError(
            "Absolute error contains non-finite values."
        )

    if not np.isfinite(risk).all():
        raise RuntimeError(
            "Risk contains non-finite values."
        )

    n = len(err)

    order = np.argsort(
        risk,
        kind="mergesort"
    )

    risk_sorted = risk[order]
    err_sorted = err[order]

    # Start index of every exact risk group.
    starts = np.r_[
        0,
        np.flatnonzero(
            risk_sorted[1:]
            !=
            risk_sorted[:-1]
        ) + 1
    ]

    ends = np.r_[
        starts[1:],
        n
    ]

    group_counts = (
        ends - starts
    ).astype(np.float64)

    group_error_sums = np.add.reduceat(
        err_sorted,
        starts
    )

    cumulative_counts = np.cumsum(
        group_counts
    )

    cumulative_error_sums = np.cumsum(
        group_error_sums
    )

    result = {}

    for coverage in coverages:

        target_count = float(
            coverage * n
        )

        if not (
            0.0
            <
            target_count
            <=
            n
        ):
            raise RuntimeError(
                "Invalid selective coverage."
            )

        group_idx = int(
            np.searchsorted(
                cumulative_counts,
                target_count,
                side="left"
            )
        )

        count_before = (
            cumulative_counts[group_idx - 1]
            if group_idx > 0
            else 0.0
        )

        error_before = (
            cumulative_error_sums[group_idx - 1]
            if group_idx > 0
            else 0.0
        )

        required_from_boundary = (
            target_count
            -
            count_before
        )

        boundary_fraction = (
            required_from_boundary
            /
            group_counts[group_idx]
        )

        selected_error_sum = (
            error_before
            +
            boundary_fraction
            *
            group_error_sums[group_idx]
        )

        selected_mae = (
            selected_error_sum
            /
            target_count
        )

        result[coverage] = float(
            selected_mae
        )

    largest_tie_fraction = float(
        np.max(group_counts)
        /
        n
    )

    return (
        result,
        largest_tie_fraction
    )


# ================================================================
# 11. CLEAN DEV CANDIDATE DIAGNOSTIC
# ================================================================

clean_abs_error_628hc3a1 = np.abs(
    clean_pred_628hc3a1
    -
    y_dev_628hc3a1
)

clean_full_mae_628hc3a1 = float(
    np.mean(clean_abs_error_628hc3a1)
)

clean_candidates_628hc3a1 = (
    build_candidate_risks_628hc3a1(
        clean_uncertainty_rebuilt_628hc3a1,
        clean_quality_628hc3a1
    )
)

clean_rows_628hc3a1 = []

for candidate_name, candidate_risk in (
    clean_candidates_628hc3a1.items()
):

    selective_maes, largest_tie = (
        tie_aware_selective_mae_628hc3a1(
            clean_abs_error_628hc3a1,
            candidate_risk
        )
    )

    spearman = safe_spearman_628hc3a1(
        candidate_risk,
        clean_abs_error_628hc3a1
    )

    clean_rows_628hc3a1.append(
        {
            "candidate":
                candidate_name,

            "full_mae":
                clean_full_mae_628hc3a1,

            "risk_error_spearman":
                spearman,

            "mae_90":
                selective_maes[0.90],

            "gain_90":
                clean_full_mae_628hc3a1
                -
                selective_maes[0.90],

            "mae_80":
                selective_maes[0.80],

            "gain_80":
                clean_full_mae_628hc3a1
                -
                selective_maes[0.80],

            "mae_70":
                selective_maes[0.70],

            "gain_70":
                clean_full_mae_628hc3a1
                -
                selective_maes[0.70],

            "risk_median":
                float(np.median(candidate_risk)),

            "largest_tie_fraction":
                largest_tie,
        }
    )


CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1 = (
    pd.DataFrame(
        clean_rows_628hc3a1
    )
)


print("\n--- CLEAN DEV CANDIDATE RESPONSE ---")
display(
    CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
)


# ================================================================
# 12. FULL 315-SCENARIO CANDIDATE EVALUATION
# ================================================================

print(
    "\n--- RUNNING 315 × CANDIDATE FUSION EVALUATION ---"
)

t0_eval_628hc3a1 = time.perf_counter()

scenario_rows_628hc3a1 = []

candidate_min_628hc3a1 = {
    name: np.inf
    for name in candidate_names_628hc3a1
}

candidate_max_628hc3a1 = {
    name: -np.inf
    for name in candidate_names_628hc3a1
}


for i in range(n_scenarios_628hc3a1):

    manifest_row = (
        manifest_628hc3a1.iloc[i]
    )

    scenario_id = (
        manifest_row["scenario_id"]
    )

    family = (
        manifest_row["family"]
    )

    severity = (
        manifest_row["severity"]
    )

    sensor = (
        manifest_row["sensor"]
    )

    prediction_i = (
        pred_matrix_628hc3a1[i]
    )

    uncertainty_i = (
        uncertainty_matrix_628hc3a1[i]
    )

    quality_i = (
        quality_matrix_628hc3a1[i]
    )

    abs_error_i = np.abs(
        prediction_i
        -
        y_dev_628hc3a1
    )

    full_mae_i = float(
        np.mean(abs_error_i)
    )

    candidate_dict_i = (
        build_candidate_risks_628hc3a1(
            uncertainty_i,
            quality_i
        )
    )

    for candidate_name, candidate_risk in (
        candidate_dict_i.items()
    ):

        candidate_min_628hc3a1[
            candidate_name
        ] = min(
            candidate_min_628hc3a1[
                candidate_name
            ],
            float(np.min(candidate_risk))
        )

        candidate_max_628hc3a1[
            candidate_name
        ] = max(
            candidate_max_628hc3a1[
                candidate_name
            ],
            float(np.max(candidate_risk))
        )

        selective_maes, largest_tie = (
            tie_aware_selective_mae_628hc3a1(
                abs_error_i,
                candidate_risk
            )
        )

        spearman = safe_spearman_628hc3a1(
            candidate_risk,
            abs_error_i
        )

        scenario_rows_628hc3a1.append(
            {
                "scenario_id":
                    scenario_id,

                "scenario_number":
                    int(
                        manifest_row[
                            "scenario_number"
                        ]
                    ),

                "family":
                    family,

                "sensor":
                    sensor,

                "severity":
                    severity,

                "candidate":
                    candidate_name,

                "full_mae":
                    full_mae_i,

                "risk_error_spearman":
                    spearman,

                "mae_90":
                    selective_maes[0.90],

                "gain_90":
                    full_mae_i
                    -
                    selective_maes[0.90],

                "mae_80":
                    selective_maes[0.80],

                "gain_80":
                    full_mae_i
                    -
                    selective_maes[0.80],

                "mae_70":
                    selective_maes[0.70],

                "gain_70":
                    full_mae_i
                    -
                    selective_maes[0.70],

                "risk_median":
                    float(
                        np.median(
                            candidate_risk
                        )
                    ),

                "risk_mean":
                    float(
                        np.mean(
                            candidate_risk
                        )
                    ),

                "largest_tie_fraction":
                    largest_tie,
            }
        )

    if (
        i == 0
        or (i + 1) % 50 == 0
        or i == n_scenarios_628hc3a1 - 1
    ):

        elapsed = (
            time.perf_counter()
            -
            t0_eval_628hc3a1
        )

        print(
            f"[{i+1:03d}/{n_scenarios_628hc3a1}] "
            f"{scenario_id} | "
            f"{family:<7} | "
            f"{severity:<6} | "
            f"full_MAE={full_mae_i:.4f} | "
            f"elapsed={elapsed:.1f}s"
        )


eval_runtime_628hc3a1 = (
    time.perf_counter()
    -
    t0_eval_628hc3a1
)


FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1 = (
    pd.DataFrame(
        scenario_rows_628hc3a1
    )
)


print(
    "\nFull candidate-response rows:",
    len(
        FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
    )
)

print(
    "Evaluation runtime (sec):",
    eval_runtime_628hc3a1
)


# ================================================================
# 13. UNCERTAINTY-BASELINE COMPARISON
# ================================================================

unc_baseline_628hc3a1 = (
    FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
        FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ][
        [
            "scenario_id",
            "mae_90",
            "mae_80",
            "mae_70",
        ]
    ]
    .rename(
        columns={
            "mae_90":
                "uncertainty_mae_90",

            "mae_80":
                "uncertainty_mae_80",

            "mae_70":
                "uncertainty_mae_70",
        }
    )
)


FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1 = (
    FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
    .merge(
        unc_baseline_628hc3a1,
        on="scenario_id",
        how="left",
        validate="many_to_one"
    )
)


for cov in (
    90,
    80,
    70
):
    FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
        f"delta_vs_uncertainty_{cov}"
    ] = (
        FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            f"uncertainty_mae_{cov}"
        ]
        -
        FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            f"mae_{cov}"
        ]
    )


# Positive:
# candidate has LOWER selective MAE than uncertainty-only.
#
# Negative:
# candidate is WORSE than uncertainty-only.


# ================================================================
# 14. CLEAN BASELINE DELTA
# ================================================================

clean_unc_row_628hc3a1 = (
    CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
        CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ]
    .iloc[0]
)

for cov in (
    90,
    80,
    70
):
    CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
        f"delta_vs_uncertainty_{cov}"
    ] = (
        float(
            clean_unc_row_628hc3a1[
                f"mae_{cov}"
            ]
        )
        -
        CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            f"mae_{cov}"
        ]
    )


print(
    "\n--- CLEAN DEV COMPARISON VS UNCERTAINTY-ONLY ---"
)

display(
    CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
        [
            "candidate",
            "risk_error_spearman",
            "mae_90",
            "gain_90",
            "mae_80",
            "gain_80",
            "mae_70",
            "gain_70",
            "delta_vs_uncertainty_80",
            "largest_tie_fraction",
        ]
    ]
)


# ================================================================
# 15. OVERALL CANDIDATE SUMMARY
# ================================================================

overall_rows_628hc3a1 = []

response_628hc3a1 = (
    FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
)


for candidate_name in candidate_names_628hc3a1:

    group = response_628hc3a1[
        response_628hc3a1[
            "candidate"
        ]
        ==
        candidate_name
    ].copy()

    delta80 = group[
        "delta_vs_uncertainty_80"
    ].to_numpy(dtype=np.float64)

    finite_spearman_count = int(
        np.isfinite(
            group[
                "risk_error_spearman"
            ].to_numpy(dtype=np.float64)
        ).sum()
    )

    tolerance = 1e-12

    better_count = int(
        np.sum(
            delta80 > tolerance
        )
    )

    equal_count = int(
        np.sum(
            np.abs(delta80)
            <= tolerance
        )
    )

    worse_count = int(
        np.sum(
            delta80 < -tolerance
        )
    )

    clean_row = (
        CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
            CLEAN_FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
                "candidate"
            ]
            ==
            candidate_name
        ]
        .iloc[0]
    )

    overall_rows_628hc3a1.append(
        {
            "candidate":
                candidate_name,

            "clean_gain_80":
                float(
                    clean_row[
                        "gain_80"
                    ]
                ),

            "clean_delta_vs_uncertainty_80":
                float(
                    clean_row[
                        "delta_vs_uncertainty_80"
                    ]
                ),

            "median_risk_error_spearman":
                float(
                    group[
                        "risk_error_spearman"
                    ].median(
                        skipna=True
                    )
                ),

            "finite_spearman_scenarios":
                finite_spearman_count,

            "median_gain_90":
                float(
                    group[
                        "gain_90"
                    ].median()
                ),

            "median_gain_80":
                float(
                    group[
                        "gain_80"
                    ].median()
                ),

            "median_gain_70":
                float(
                    group[
                        "gain_70"
                    ].median()
                ),

            "mean_gain_80":
                float(
                    group[
                        "gain_80"
                    ].mean()
                ),

            "median_delta_vs_uncertainty_80":
                float(
                    group[
                        "delta_vs_uncertainty_80"
                    ].median()
                ),

            "mean_delta_vs_uncertainty_80":
                float(
                    group[
                        "delta_vs_uncertainty_80"
                    ].mean()
                ),

            "better_than_uncertainty_80":
                better_count,

            "equal_to_uncertainty_80":
                equal_count,

            "worse_than_uncertainty_80":
                worse_count,

            "median_largest_tie_fraction":
                float(
                    group[
                        "largest_tie_fraction"
                    ].median()
                ),

            "minimum_observed_risk":
                float(
                    candidate_min_628hc3a1[
                        candidate_name
                    ]
                ),

            "maximum_observed_risk":
                float(
                    candidate_max_628hc3a1[
                        candidate_name
                    ]
                ),
        }
    )


FD002_COMBINED_CANDIDATE_SUMMARY_628HC3A1 = (
    pd.DataFrame(
        overall_rows_628hc3a1
    )
    .sort_values(
        by=[
            "median_delta_vs_uncertainty_80",
            "mean_delta_vs_uncertainty_80",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n--- OVERALL CANDIDATE SUMMARY ---")

display(
    FD002_COMBINED_CANDIDATE_SUMMARY_628HC3A1
)


# ================================================================
# 16. FAMILY × SEVERITY SUMMARY
# ================================================================

family_severity_summary_628hc3a1 = (
    response_628hc3a1
    .groupby(
        [
            "candidate",
            "family",
            "severity",
        ],
        observed=True
    )
    .agg(
        scenarios=(
            "scenario_id",
            "size"
        ),

        median_full_mae=(
            "full_mae",
            "median"
        ),

        median_risk_error_spearman=(
            "risk_error_spearman",
            "median"
        ),

        median_gain_90=(
            "gain_90",
            "median"
        ),

        median_gain_80=(
            "gain_80",
            "median"
        ),

        median_gain_70=(
            "gain_70",
            "median"
        ),

        mean_gain_80=(
            "gain_80",
            "mean"
        ),

        median_delta_vs_uncertainty_80=(
            "delta_vs_uncertainty_80",
            "median"
        ),

        mean_delta_vs_uncertainty_80=(
            "delta_vs_uncertainty_80",
            "mean"
        ),

        median_largest_tie_fraction=(
            "largest_tie_fraction",
            "median"
        ),
    )
    .reset_index()
)


FD002_COMBINED_FAMILY_SEVERITY_SUMMARY_628HC3A1 = (
    family_severity_summary_628hc3a1
)


print(
    "\n--- FAMILY × SEVERITY: "
    "MEDIAN Δ VS UNCERTAINTY @ 80% COVERAGE ---"
)

family_severity_delta_pivot_628hc3a1 = (
    family_severity_summary_628hc3a1
    .pivot_table(
        index=[
            "family",
            "severity"
        ],
        columns="candidate",
        values="median_delta_vs_uncertainty_80",
        aggfunc="first"
    )
)


display(
    family_severity_delta_pivot_628hc3a1
)


print(
    "\nInterpretation:"
)
print(
    "  positive value = candidate improves over "
    "uncertainty-only"
)
print(
    "  zero           = same as uncertainty-only"
)
print(
    "  negative value = candidate is worse than "
    "uncertainty-only"
)


# ================================================================
# 17. FAMILY × SEVERITY ABSOLUTE SELECTIVE GAIN
# ================================================================

print(
    "\n--- FAMILY × SEVERITY: "
    "MEDIAN ABSOLUTE MAE GAIN @ 80% COVERAGE ---"
)

family_severity_gain_pivot_628hc3a1 = (
    family_severity_summary_628hc3a1
    .pivot_table(
        index=[
            "family",
            "severity"
        ],
        columns="candidate",
        values="median_gain_80",
        aggfunc="first"
    )
)


display(
    family_severity_gain_pivot_628hc3a1
)


# ================================================================
# 18. BEST / WORST DELTAS RELATIVE TO UNCERTAINTY
# ================================================================

non_baseline_628hc3a1 = (
    response_628hc3a1[
        response_628hc3a1[
            "candidate"
        ]
        !=
        "uncertainty_only"
    ]
    .copy()
)


print(
    "\n--- TOP 20 IMPROVEMENTS VS UNCERTAINTY "
    "@ 80% COVERAGE ---"
)

display(
    non_baseline_628hc3a1
    .sort_values(
        "delta_vs_uncertainty_80",
        ascending=False
    )
    [
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "candidate",
            "full_mae",
            "mae_80",
            "gain_80",
            "delta_vs_uncertainty_80",
            "risk_error_spearman",
            "largest_tie_fraction",
        ]
    ]
    .head(20)
)


print(
    "\n--- WORST 20 DEGRADATIONS VS UNCERTAINTY "
    "@ 80% COVERAGE ---"
)

display(
    non_baseline_628hc3a1
    .sort_values(
        "delta_vs_uncertainty_80",
        ascending=True
    )
    [
        [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "candidate",
            "full_mae",
            "mae_80",
            "gain_80",
            "delta_vs_uncertainty_80",
            "risk_error_spearman",
            "largest_tie_fraction",
        ]
    ]
    .head(20)
)


# ================================================================
# 19. FINAL AUDIT
# ================================================================

expected_response_rows_628hc3a1 = (
    n_scenarios_628hc3a1
    *
    len(candidate_names_628hc3a1)
)

metric_columns_628hc3a1 = [
    "full_mae",
    "mae_90",
    "gain_90",
    "mae_80",
    "gain_80",
    "mae_70",
    "gain_70",
    "risk_median",
    "risk_mean",
    "largest_tie_fraction",
    "delta_vs_uncertainty_90",
    "delta_vs_uncertainty_80",
    "delta_vs_uncertainty_70",
]


audit_rows_628hc3a1 = [
    {
        "check":
            "scenario_count_315",
        "passed":
            n_scenarios_628hc3a1 == 315,
    },

    {
        "check":
            "candidate_count_6",
        "passed":
            len(candidate_names_628hc3a1) == 6,
    },

    {
        "check":
            "response_row_count_exact",
        "passed":
            len(
                FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
            )
            ==
            expected_response_rows_628hc3a1,
    },

    {
        "check":
            "uncertainty_matrix_shape_exact",
        "passed":
            uncertainty_matrix_628hc3a1.shape
            ==
            expected_shape_628hc3a1,
    },

    {
        "check":
            "uncertainty_matrix_finite",
        "passed":
            bool(
                np.isfinite(
                    uncertainty_matrix_628hc3a1
                ).all()
            ),
    },

    {
        "check":
            "quality_matrix_shape_exact",
        "passed":
            quality_matrix_628hc3a1.shape
            ==
            expected_shape_628hc3a1,
    },

    {
        "check":
            "quality_matrix_finite",
        "passed":
            bool(
                np.isfinite(
                    quality_matrix_628hc3a1
                ).all()
            ),
    },

    {
        "check":
            "clean_uncertainty_chain_exact",
        "passed":
            clean_uncertainty_exact_628hc3a1,
    },

    {
        "check":
            "selective_metrics_finite",
        "passed":
            bool(
                np.isfinite(
                    FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1[
                        metric_columns_628hc3a1
                    ].to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
    },
]


FD002_COMBINED_CANDIDATE_AUDIT_628HC3A1 = (
    pd.DataFrame(
        audit_rows_628hc3a1
    )
)


print("\n--- FINAL CANDIDATE-EVALUATION AUDIT ---")
display(
    FD002_COMBINED_CANDIDATE_AUDIT_628HC3A1
)

all_audit_passed_628hc3a1 = bool(
    FD002_COMBINED_CANDIDATE_AUDIT_628HC3A1[
        "passed"
    ].all()
)

if not all_audit_passed_628hc3a1:
    raise RuntimeError(
        "C3-A1 final audit failed. "
        "Do not continue."
    )


# ================================================================
# 20. FROZEN IN-MEMORY CONTRACT FOR NEXT STEP
# ================================================================

FD002_COMBINED_FUSION_EVALUATION_CONTRACT_628HC3A1 = {
    "step":
        "6.28H-C3-A1",

    "purpose":
        "DEV-only candidate reliability fusion comparison",

    "scenarios":
        int(n_scenarios_628hc3a1),

    "dev_windows":
        int(n_dev_628hc3a1),

    "candidate_rules":
        tuple(candidate_names_628hc3a1),

    "coverages":
        tuple(COVERAGES_628HC3A1),

    "risk_direction":
        "lower_risk_accepted_first",

    "boundary_tie_policy":
        "expected_error_within_exact_tied_risk_group",

    "quality_gate_source":
        "FIT sensor-quality q95",

    "quality_gate_q95":
        float(fit_q95_628hc3a1),

    "uncertainty_reference":
        "frozen FD002 clean reliability chain",

    "sensor_quality_reference":
        "FIT-calibrated label-free sensor-quality risk",

    "clean_uncertainty_chain_exact":
        bool(clean_uncertainty_exact_628hc3a1),

    "all_audit_checks_passed":
        bool(all_audit_passed_628hc3a1),

    "combination_rule_selected":
        False,

    "combination_weight_tuned":
        False,

    "abstention_threshold_selected":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# ================================================================
# 21. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 82)
print("STEP 6.28H-C3-A1 FINAL SUMMARY")
print("=" * 82)

print(
    "Scenarios evaluated              :",
    n_scenarios_628hc3a1
)

print(
    "DEV windows/scenario             :",
    n_dev_628hc3a1
)

print(
    "Candidate fusion rules           :",
    len(candidate_names_628hc3a1)
)

print(
    "Candidate-response rows          :",
    len(
        FD002_COMBINED_CANDIDATE_RESPONSE_628HC3A1
    )
)

print(
    "Clean uncertainty chain exact    :",
    clean_uncertainty_exact_628hc3a1
)

print(
    "FIT quality q95                  :",
    fit_q95_628hc3a1
)

print(
    "Uncertainty reconstruction sec   :",
    unc_runtime_628hc3a1
)

print(
    "Candidate evaluation sec         :",
    eval_runtime_628hc3a1
)

print(
    "All audit checks passed          :",
    all_audit_passed_628hc3a1
)

print(
    "Combination rule selected        :",
    False
)

print(
    "Combination weight tuned         :",
    False
)

print(
    "Abstention threshold selected    :",
    False
)

print(
    "DEV RUL labels used              :",
    True
)

print(
    "Official TEST labels read        :",
    False
)

print(
    "TEST scored                      :",
    False
)

print(
    "Model retrained                  :",
    False
)

print(
    "Files written                    :",
    False
)

print("\n=== END STEP 6.28H-C3-A1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A1-R1
# LOCATE EXACT C3-A0-R1 UNCERTAINTY-RISK MATRIX
# INSPECTION ONLY — NOTHING MODIFIED
# ================================================================

import numpy as np
import pandas as pd

print("=" * 82)
print("STEP 6.28H-C3-A1-R1 — LOCATE FROZEN C3-A0-R1 UNCERTAINTY MATRIX")
print("=" * 82)

rows_628hc3a1r1 = []

for name, obj in list(globals().items()):

    lname = name.lower()

    if "uncert" not in lname:
        continue

    try:
        shape = tuple(obj.shape) if hasattr(obj, "shape") else None
    except Exception:
        shape = None

    try:
        length = len(obj)
    except Exception:
        length = None

    if (
        shape == (315, 8367)
        or
        (
            "risk" in lname
            and shape is not None
        )
    ):
        rows_628hc3a1r1.append(
            {
                "object_name": name,
                "type": type(obj).__name__,
                "shape": shape,
                "length": length,
            }
        )

UNCERTAINTY_MATRIX_CANDIDATES_628HC3A1R1 = pd.DataFrame(
    rows_628hc3a1r1
)

print("\n--- UNCERTAINTY / RISK MATRIX CANDIDATES ---")

if len(UNCERTAINTY_MATRIX_CANDIDATES_628HC3A1R1) == 0:
    print("No matching candidates found.")
else:
    display(
        UNCERTAINTY_MATRIX_CANDIDATES_628HC3A1R1
        .sort_values("object_name")
        .reset_index(drop=True)
    )

print("\nNo model executed       : True")
print("No TEST labels read    : True")
print("No threshold selected  : True")
print("No files written       : True")

print("\n=== END STEP 6.28H-C3-A1-R1 ===")

In [ ]:
# =====================================================================
# STEP 6.28H-C3-A1
# CANDIDATE COMBINED-RELIABILITY FUSION EVALUATION
#
# IMPORTANT:
# - Reuses the EXACT frozen uncertainty-risk matrix from C3-A0-R1.
# - Reuses the frozen sensor-quality matrix from C2-C.
# - DEV RUL labels are used ONLY for development evaluation.
# - No official TEST labels are read.
# - No abstention threshold is selected.
# - No final fusion rule is selected in this step.
# - No model is retrained or executed.
# - No files are written.
# =====================================================================

import numpy as np
import pandas as pd
import time
import warnings

from scipy.stats import spearmanr


# =====================================================================
# 1. HEADER
# =====================================================================

print("=" * 94)
print("STEP 6.28H-C3-A1 — CANDIDATE COMBINED-RELIABILITY FUSION EVALUATION")
print("=" * 94)


# =====================================================================
# 2. REQUIRED OBJECT CONTRACT
# =====================================================================

required_objects_628hc3a1 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",

    # Exact frozen uncertainty branch
    "uncertainty_risk_matrix_628hc3a0r1",
    "clean_uncertainty_risk_628hc3a0r1",

    # Original clean uncertainty reference for exact audit
    "FD002_DEV_SELECTIVE_RISK_628FCR2",

    # Sensor-quality branch
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc3a1 = []

for name in required_objects_628hc3a1:
    exists = name in globals()
    print(f"{name:<55}: {exists}")

    if not exists:
        missing_628hc3a1.append(name)

if missing_628hc3a1:
    raise RuntimeError(
        "Missing required objects for C3-A1: "
        + ", ".join(missing_628hc3a1)
    )


# =====================================================================
# 3. RECOVER FROZEN INPUTS
# =====================================================================

manifest_628hc3a1 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

pred_matrix_628hc3a1 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

y_dev_628hc3a1 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_pred_628hc3a1 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64
).reshape(-1)

# ---------------------------------------------------------------------
# EXACT uncertainty branch from C3-A0-R1
# ---------------------------------------------------------------------

u_matrix_628hc3a1 = np.asarray(
    uncertainty_risk_matrix_628hc3a0r1,
    dtype=np.float64
)

u_clean_628hc3a1 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64
).reshape(-1)

# ---------------------------------------------------------------------
# Frozen sensor-quality branch
# ---------------------------------------------------------------------

q_matrix_628hc3a1 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

clean_quality_object_628hc3a1 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

if not isinstance(clean_quality_object_628hc3a1, dict):
    raise TypeError(
        "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
        "must be a dictionary."
    )

if "sensor_quality_risk" not in clean_quality_object_628hc3a1:
    raise KeyError(
        "Dictionary does not contain key 'sensor_quality_risk'."
    )

q_clean_628hc3a1 = np.asarray(
    clean_quality_object_628hc3a1["sensor_quality_risk"],
    dtype=np.float64
).reshape(-1)

quality_q95_marker_628hc3a1 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)


# =====================================================================
# 4. POPULATION CONTRACT
# =====================================================================

n_scenarios_628hc3a1 = len(manifest_628hc3a1)
n_dev_628hc3a1 = len(y_dev_628hc3a1)

expected_matrix_shape_628hc3a1 = (
    n_scenarios_628hc3a1,
    n_dev_628hc3a1,
)

print("\n--- POPULATION CONTRACT ---")
print("Scenarios                 :", n_scenarios_628hc3a1)
print("DEV windows               :", n_dev_628hc3a1)
print("Prediction matrix         :", pred_matrix_628hc3a1.shape)
print("Uncertainty-risk matrix   :", u_matrix_628hc3a1.shape)
print("Sensor-quality matrix     :", q_matrix_628hc3a1.shape)
print("Clean uncertainty risk    :", u_clean_628hc3a1.shape)
print("Clean sensor-quality risk :", q_clean_628hc3a1.shape)
print("Quality FIT q95 marker    :", quality_q95_marker_628hc3a1)

if n_scenarios_628hc3a1 != 315:
    raise RuntimeError(
        f"Expected 315 scenarios; found {n_scenarios_628hc3a1}."
    )

if pred_matrix_628hc3a1.shape != expected_matrix_shape_628hc3a1:
    raise RuntimeError("Prediction matrix shape mismatch.")

if u_matrix_628hc3a1.shape != expected_matrix_shape_628hc3a1:
    raise RuntimeError("Uncertainty-risk matrix shape mismatch.")

if q_matrix_628hc3a1.shape != expected_matrix_shape_628hc3a1:
    raise RuntimeError("Sensor-quality matrix shape mismatch.")

if clean_pred_628hc3a1.shape != (n_dev_628hc3a1,):
    raise RuntimeError("Clean prediction vector shape mismatch.")

if u_clean_628hc3a1.shape != (n_dev_628hc3a1,):
    raise RuntimeError("Clean uncertainty vector shape mismatch.")

if q_clean_628hc3a1.shape != (n_dev_628hc3a1,):
    raise RuntimeError("Clean quality-risk vector shape mismatch.")


# =====================================================================
# 5. NUMERICAL INTEGRITY
# =====================================================================

numerical_arrays_628hc3a1 = {
    "prediction_matrix": pred_matrix_628hc3a1,
    "uncertainty_matrix": u_matrix_628hc3a1,
    "quality_matrix": q_matrix_628hc3a1,
    "y_dev": y_dev_628hc3a1,
    "clean_prediction": clean_pred_628hc3a1,
    "clean_uncertainty": u_clean_628hc3a1,
    "clean_quality": q_clean_628hc3a1,
}

print("\n--- NUMERICAL INTEGRITY ---")

for name, arr in numerical_arrays_628hc3a1.items():

    finite = bool(np.all(np.isfinite(arr)))

    print(f"{name:<24}: finite={finite}")

    if not finite:
        raise RuntimeError(
            f"Non-finite values detected in {name}."
        )

risk_arrays_628hc3a1 = {
    "uncertainty_matrix": u_matrix_628hc3a1,
    "quality_matrix": q_matrix_628hc3a1,
    "clean_uncertainty": u_clean_628hc3a1,
    "clean_quality": q_clean_628hc3a1,
}

for name, arr in risk_arrays_628hc3a1.items():

    within = bool(
        np.all((arr >= -1e-12) & (arr <= 1.0 + 1e-12))
    )

    print(f"{name:<24}: within_[0,1]={within}")

    if not within:
        raise RuntimeError(
            f"{name} contains values outside [0,1]."
        )


# =====================================================================
# 6. EXACT FROZEN-CHAIN AUDIT
#
# C3-A0-R1 already established the exact clean uncertainty chain.
# We verify that exact object again before fusion.
# =====================================================================

stored_clean_uncertainty_628hc3a1 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)

clean_u_max_error_628hc3a1 = float(
    np.max(
        np.abs(
            u_clean_628hc3a1
            - stored_clean_uncertainty_628hc3a1
        )
    )
)

clean_u_exact_628hc3a1 = bool(
    clean_u_max_error_628hc3a1 <= 1e-12
)

print("\n--- EXACT FROZEN UNCERTAINTY-CHAIN AUDIT ---")
print(
    "Clean uncertainty maximum absolute error:",
    clean_u_max_error_628hc3a1,
)
print(
    "Exact <= 1e-12                         :",
    clean_u_exact_628hc3a1,
)

if not clean_u_exact_628hc3a1:
    raise RuntimeError(
        "C3-A0-R1 clean uncertainty object does not match "
        "the frozen uncertainty reference."
    )


# Optional corrupted-matrix consistency check against C1-D
if (
    "FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D"
    in globals()
):

    old_u_matrix_628hc3a1 = np.asarray(
        FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D,
        dtype=np.float64
    )

    if old_u_matrix_628hc3a1.shape == u_matrix_628hc3a1.shape:

        corrupted_u_max_error_628hc3a1 = float(
            np.max(
                np.abs(
                    old_u_matrix_628hc3a1
                    - u_matrix_628hc3a1
                )
            )
        )

        print(
            "C1-D vs C3-A0-R1 corrupted-U max error  :",
            corrupted_u_max_error_628hc3a1,
        )

    else:
        corrupted_u_max_error_628hc3a1 = np.nan

else:
    corrupted_u_max_error_628hc3a1 = np.nan


# =====================================================================
# 7. SELECTIVE-RISK EVALUATION POLICY
#
# Lower risk is accepted first.
#
# Tie handling:
# If the coverage boundary cuts through a group having exactly the
# same risk value, expected absolute error within that tied group is
# used. This avoids arbitrary ordering of tied samples.
# =====================================================================

COVERAGES_628HC3A1 = (0.90, 0.80, 0.70)


def tie_aware_selective_mae_628hc3a1(
    absolute_errors,
    risk_values,
    coverages=COVERAGES_628HC3A1,
):
    """
    Expected selective MAE under exact-risk ties.

    Lower risk is accepted first.

    At the boundary, if only part of a tied risk group is required,
    the expected error of that tied group is used.
    """

    e = np.asarray(
        absolute_errors,
        dtype=np.float64
    ).reshape(-1)

    r = np.asarray(
        risk_values,
        dtype=np.float64
    ).reshape(-1)

    if len(e) != len(r):
        raise ValueError(
            "Error and risk vectors must have the same length."
        )

    if not np.all(np.isfinite(e)):
        raise ValueError("Absolute-error vector is not finite.")

    if not np.all(np.isfinite(r)):
        raise ValueError("Risk vector is not finite.")

    n = len(e)

    order = np.argsort(
        r,
        kind="mergesort"
    )

    rs = r[order]
    es = e[order]

    # Exact tied-risk groups
    group_starts = np.r_[
        0,
        np.flatnonzero(rs[1:] != rs[:-1]) + 1
    ]

    group_ends = np.r_[
        group_starts[1:],
        n
    ]

    group_counts = (
        group_ends - group_starts
    ).astype(np.int64)

    group_error_sums = np.add.reduceat(
        es,
        group_starts
    )

    cumulative_counts = np.cumsum(
        group_counts
    )

    cumulative_error_sums = np.cumsum(
        group_error_sums
    )

    largest_tie_fraction = float(
        np.max(group_counts) / n
    )

    output = {
        "largest_tie_fraction": largest_tie_fraction,
        "unique_risk_values": int(len(group_counts)),
    }

    for coverage in coverages:

        # nearest whole observation to requested coverage
        k = int(np.rint(float(coverage) * n))

        k = max(
            1,
            min(k, n)
        )

        boundary_group = int(
            np.searchsorted(
                cumulative_counts,
                k,
                side="left"
            )
        )

        if boundary_group == 0:
            count_before = 0
            sum_before = 0.0
        else:
            count_before = int(
                cumulative_counts[
                    boundary_group - 1
                ]
            )

            sum_before = float(
                cumulative_error_sums[
                    boundary_group - 1
                ]
            )

        required_from_boundary = (
            k - count_before
        )

        boundary_count = int(
            group_counts[
                boundary_group
            ]
        )

        boundary_fraction = float(
            required_from_boundary
            / boundary_count
        )

        expected_boundary_error = float(
            boundary_fraction
            * group_error_sums[
                boundary_group
            ]
        )

        selective_mae = float(
            (
                sum_before
                + expected_boundary_error
            )
            / k
        )

        pct = int(
            round(
                100 * float(coverage)
            )
        )

        output[
            f"mae_at_{pct}pct_coverage"
        ] = selective_mae

    return output


def safe_spearman_628hc3a1(
    x,
    y,
):
    """
    Spearman correlation with constant-vector protection.
    """

    x = np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)

    y = np.asarray(
        y,
        dtype=np.float64
    ).reshape(-1)

    if len(x) != len(y):
        raise ValueError(
            "Spearman vectors have different lengths."
        )

    if np.unique(x).size < 2:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        rho = spearmanr(
            x,
            y
        ).correlation

    if rho is None:
        return np.nan

    return float(rho)


print("\n--- SELECTIVE-RISK EVALUATION POLICY ---")
print("Risk direction       : lower risk accepted first")
print("Coverages            :", COVERAGES_628HC3A1)
print(
    "Boundary tie policy : expected error within exact tied-risk group"
)
print("Threshold selection  : False")


# =====================================================================
# 8. QUALITY-EXCESS TRANSFORMATION
#
# This branch activates sensor-quality evidence only when quality risk
# exceeds the FIT-only q95 diagnostic marker.
#
# No RUL labels are used in this transformation.
# =====================================================================

def quality_excess_628hc3a1(q):

    q = np.asarray(
        q,
        dtype=np.float64
    )

    denominator = (
        1.0
        - quality_q95_marker_628hc3a1
    )

    if denominator <= 0.0:
        raise RuntimeError(
            "Invalid FIT quality q95 marker."
        )

    excess = (
        q
        - quality_q95_marker_628hc3a1
    ) / denominator

    return np.clip(
        excess,
        0.0,
        1.0
    )


# =====================================================================
# 9. FUSION CANDIDATES
#
# Two single-branch references:
#   uncertainty_only
#   quality_only
#
# Raw fusion candidates:
#   mean_50_50
#   weighted_u75_q25
#   weighted_u25_q75
#   maximum
#   noisy_or
#
# FIT-q95 gated quality candidates:
#   excess_or_025
#   excess_or_050
#   excess_or_100
#
# No candidate is selected here.
# =====================================================================

FUSION_CANDIDATES_628HC3A1 = [
    "uncertainty_only",
    "quality_only",
    "mean_50_50",
    "weighted_u75_q25",
    "weighted_u25_q75",
    "maximum",
    "noisy_or",
    "excess_or_025",
    "excess_or_050",
    "excess_or_100",
]


def fuse_risk_628hc3a1(
    candidate,
    u,
    q,
):

    u = np.asarray(
        u,
        dtype=np.float64
    )

    q = np.asarray(
        q,
        dtype=np.float64
    )

    if u.shape != q.shape:
        raise ValueError(
            "Uncertainty and quality arrays have different shapes."
        )

    if candidate == "uncertainty_only":

        result = u

    elif candidate == "quality_only":

        result = q

    elif candidate == "mean_50_50":

        result = (
            0.50 * u
            + 0.50 * q
        )

    elif candidate == "weighted_u75_q25":

        result = (
            0.75 * u
            + 0.25 * q
        )

    elif candidate == "weighted_u25_q75":

        result = (
            0.25 * u
            + 0.75 * q
        )

    elif candidate == "maximum":

        result = np.maximum(
            u,
            q
        )

    elif candidate == "noisy_or":

        result = (
            1.0
            - (1.0 - u)
            * (1.0 - q)
        )

    elif candidate in {
        "excess_or_025",
        "excess_or_050",
        "excess_or_100",
    }:

        gamma_map = {
            "excess_or_025": 0.25,
            "excess_or_050": 0.50,
            "excess_or_100": 1.00,
        }

        gamma = gamma_map[
            candidate
        ]

        q_excess = quality_excess_628hc3a1(
            q
        )

        # Quality contributes only beyond FIT q95.
        result = (
            u
            + (1.0 - u)
            * gamma
            * q_excess
        )

    else:

        raise ValueError(
            f"Unknown fusion candidate: {candidate}"
        )

    result = np.clip(
        result,
        0.0,
        1.0
    )

    return result


print("\n--- FUSION CANDIDATES ---")

for candidate in FUSION_CANDIDATES_628HC3A1:
    print(candidate)

print(
    "\nTotal candidate rules:",
    len(FUSION_CANDIDATES_628HC3A1),
)


# =====================================================================
# 10. CLEAN DEV BASELINE
# =====================================================================

clean_abs_error_628hc3a1 = np.abs(
    clean_pred_628hc3a1
    - y_dev_628hc3a1
)

clean_full_mae_628hc3a1 = float(
    np.mean(
        clean_abs_error_628hc3a1
    )
)

print("\n--- CLEAN DEV BASELINE ---")
print(
    "Clean full MAE:",
    clean_full_mae_628hc3a1
)


# =====================================================================
# 11. EVALUATE CLEAN DEV FOR EVERY CANDIDATE
# =====================================================================

clean_candidate_rows_628hc3a1 = []

for candidate in FUSION_CANDIDATES_628HC3A1:

    fused_clean = fuse_risk_628hc3a1(
        candidate,
        u_clean_628hc3a1,
        q_clean_628hc3a1,
    )

    clean_eval = (
        tie_aware_selective_mae_628hc3a1(
            clean_abs_error_628hc3a1,
            fused_clean,
        )
    )

    clean_rho = safe_spearman_628hc3a1(
        fused_clean,
        clean_abs_error_628hc3a1,
    )

    row = {
        "candidate": candidate,
        "risk_error_spearman": clean_rho,
        "risk_median": float(
            np.median(fused_clean)
        ),
        "risk_mean": float(
            np.mean(fused_clean)
        ),
        "largest_tie_fraction": float(
            clean_eval[
                "largest_tie_fraction"
            ]
        ),
        "unique_risk_values": int(
            clean_eval[
                "unique_risk_values"
            ]
        ),
    }

    for coverage in COVERAGES_628HC3A1:

        pct = int(
            round(
                100 * coverage
            )
        )

        mae_key = (
            f"mae_at_{pct}pct_coverage"
        )

        mae_value = float(
            clean_eval[
                mae_key
            ]
        )

        row[
            mae_key
        ] = mae_value

        row[
            f"gain_at_{pct}pct_coverage"
        ] = float(
            clean_full_mae_628hc3a1
            - mae_value
        )

    clean_candidate_rows_628hc3a1.append(
        row
    )


FD002_CLEAN_FUSION_CANDIDATE_RESULTS_628HC3A1 = (
    pd.DataFrame(
        clean_candidate_rows_628hc3a1
    )
)

print(
    "\n--- CLEAN DEV CANDIDATE RESPONSE ---"
)

display(
    FD002_CLEAN_FUSION_CANDIDATE_RESULTS_628HC3A1
    .copy()
)


# =====================================================================
# 12. FULL 315-SCENARIO CANDIDATE EVALUATION
# =====================================================================

start_time_628hc3a1 = time.perf_counter()

scenario_candidate_rows_628hc3a1 = []

print(
    "\n--- RUNNING FUSION CANDIDATES "
    "OVER 315 CORRUPTED DEV SCENARIOS ---"
)

for candidate_index, candidate in enumerate(
    FUSION_CANDIDATES_628HC3A1,
    start=1,
):

    candidate_start = time.perf_counter()

    fused_matrix = fuse_risk_628hc3a1(
        candidate,
        u_matrix_628hc3a1,
        q_matrix_628hc3a1,
    )

    if fused_matrix.shape != expected_matrix_shape_628hc3a1:
        raise RuntimeError(
            f"{candidate}: fused matrix shape mismatch."
        )

    if not np.all(np.isfinite(fused_matrix)):
        raise RuntimeError(
            f"{candidate}: non-finite fused risk values."
        )

    if not np.all(
        (fused_matrix >= -1e-12)
        &
        (fused_matrix <= 1.0 + 1e-12)
    ):
        raise RuntimeError(
            f"{candidate}: fused risk outside [0,1]."
        )

    for scenario_index in range(
        n_scenarios_628hc3a1
    ):

        manifest_row = manifest_628hc3a1.iloc[
            scenario_index
        ]

        pred = pred_matrix_628hc3a1[
            scenario_index
        ]

        risk = fused_matrix[
            scenario_index
        ]

        abs_error = np.abs(
            pred
            - y_dev_628hc3a1
        )

        full_mae = float(
            np.mean(abs_error)
        )

        selective = (
            tie_aware_selective_mae_628hc3a1(
                abs_error,
                risk,
            )
        )

        rho = safe_spearman_628hc3a1(
            risk,
            abs_error,
        )

        result_row = {
            "candidate": candidate,
            "scenario_id": manifest_row[
                "scenario_id"
            ],
            "scenario_number": int(
                manifest_row[
                    "scenario_number"
                ]
            ),
            "family": manifest_row[
                "family"
            ],
            "sensor": manifest_row[
                "sensor"
            ],
            "severity": manifest_row[
                "severity"
            ],
            "full_mae": full_mae,
            "risk_error_spearman": rho,
            "risk_median": float(
                np.median(risk)
            ),
            "risk_mean": float(
                np.mean(risk)
            ),
            "largest_tie_fraction": float(
                selective[
                    "largest_tie_fraction"
                ]
            ),
            "unique_risk_values": int(
                selective[
                    "unique_risk_values"
                ]
            ),
        }

        for coverage in COVERAGES_628HC3A1:

            pct = int(
                round(
                    100 * coverage
                )
            )

            mae_key = (
                f"mae_at_{pct}pct_coverage"
            )

            mae_value = float(
                selective[
                    mae_key
                ]
            )

            result_row[
                mae_key
            ] = mae_value

            result_row[
                f"gain_at_{pct}pct_coverage"
            ] = float(
                full_mae
                - mae_value
            )

        scenario_candidate_rows_628hc3a1.append(
            result_row
        )

    elapsed_candidate = (
        time.perf_counter()
        - candidate_start
    )

    print(
        f"[{candidate_index:02d}/"
        f"{len(FUSION_CANDIDATES_628HC3A1):02d}] "
        f"{candidate:<20} "
        f"| elapsed={elapsed_candidate:.2f}s"
    )


FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1 = (
    pd.DataFrame(
        scenario_candidate_rows_628hc3a1
    )
)

total_runtime_628hc3a1 = (
    time.perf_counter()
    - start_time_628hc3a1
)

print(
    "\nFull scenario-candidate rows:",
    len(
        FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1
    )
)

print(
    "Evaluation runtime (sec):",
    total_runtime_628hc3a1
)


# =====================================================================
# 13. CANDIDATE SUMMARY
# =====================================================================

summary_rows_628hc3a1 = []

scenario_results_628hc3a1 = (
    FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1
)

for candidate in FUSION_CANDIDATES_628HC3A1:

    sub = scenario_results_628hc3a1[
        scenario_results_628hc3a1[
            "candidate"
        ] == candidate
    ].copy()

    stuck_sub = sub[
        sub["family"] == "stuck"
    ]

    non_stuck_sub = sub[
        sub["family"] != "stuck"
    ]

    gain80 = sub[
        "gain_at_80pct_coverage"
    ].to_numpy(
        dtype=np.float64
    )

    rho_values = sub[
        "risk_error_spearman"
    ].to_numpy(
        dtype=np.float64
    )

    finite_rho = rho_values[
        np.isfinite(
            rho_values
        )
    ]

    clean_row = (
        FD002_CLEAN_FUSION_CANDIDATE_RESULTS_628HC3A1[
            FD002_CLEAN_FUSION_CANDIDATE_RESULTS_628HC3A1[
                "candidate"
            ] == candidate
        ]
        .iloc[0]
    )

    summary_rows_628hc3a1.append(
        {
            "candidate": candidate,

            "clean_spearman": float(
                clean_row[
                    "risk_error_spearman"
                ]
            )
            if np.isfinite(
                clean_row[
                    "risk_error_spearman"
                ]
            )
            else np.nan,

            "clean_gain_90": float(
                clean_row[
                    "gain_at_90pct_coverage"
                ]
            ),

            "clean_gain_80": float(
                clean_row[
                    "gain_at_80pct_coverage"
                ]
            ),

            "clean_gain_70": float(
                clean_row[
                    "gain_at_70pct_coverage"
                ]
            ),

            "median_scenario_spearman": (
                float(
                    np.median(
                        finite_rho
                    )
                )
                if len(finite_rho)
                else np.nan
            ),

            "finite_spearman_scenarios": int(
                len(finite_rho)
            ),

            "median_gain_90": float(
                np.median(
                    sub[
                        "gain_at_90pct_coverage"
                    ]
                )
            ),

            "median_gain_80": float(
                np.median(
                    sub[
                        "gain_at_80pct_coverage"
                    ]
                )
            ),

            "median_gain_70": float(
                np.median(
                    sub[
                        "gain_at_70pct_coverage"
                    ]
                )
            ),

            "mean_gain_80": float(
                np.mean(gain80)
            ),

            "q10_gain_80": float(
                np.quantile(
                    gain80,
                    0.10
                )
            ),

            "minimum_gain_80": float(
                np.min(gain80)
            ),

            "positive_gain_80": int(
                np.sum(
                    gain80 > 1e-12
                )
            ),

            "zero_gain_80": int(
                np.sum(
                    np.abs(
                        gain80
                    ) <= 1e-12
                )
            ),

            "negative_gain_80": int(
                np.sum(
                    gain80 < -1e-12
                )
            ),

            "median_stuck_gain_80": float(
                np.median(
                    stuck_sub[
                        "gain_at_80pct_coverage"
                    ]
                )
            ),

            "median_non_stuck_gain_80": float(
                np.median(
                    non_stuck_sub[
                        "gain_at_80pct_coverage"
                    ]
                )
            ),

            "median_largest_tie_fraction": float(
                np.median(
                    sub[
                        "largest_tie_fraction"
                    ]
                )
            ),
        }
    )


FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1 = (
    pd.DataFrame(
        summary_rows_628hc3a1
    )
)


# =====================================================================
# 14. COMPARE AGAINST UNCERTAINTY-ONLY REFERENCE
# =====================================================================

uncertainty_reference_row_628hc3a1 = (
    FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
        FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
            "candidate"
        ] == "uncertainty_only"
    ]
    .iloc[0]
)

uncertainty_reference_gain80_628hc3a1 = float(
    uncertainty_reference_row_628hc3a1[
        "median_gain_80"
    ]
)

FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
    "delta_median_gain80_vs_uncertainty"
] = (
    FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
        "median_gain_80"
    ]
    - uncertainty_reference_gain80_628hc3a1
)

FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
    "is_combined_candidate"
] = ~FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1[
    "candidate"
].isin(
    [
        "uncertainty_only",
        "quality_only",
    ]
)


# =====================================================================
# 15. PRIMARY DEVELOPMENT RANKING
#
# IMPORTANT:
# This is a ranking only.
#
# We DO NOT freeze/select the winner in this step.
#
# Primary diagnostic:
#   median MAE gain at 80% coverage across the 315 DEV scenarios.
#
# Secondary fields are shown for robustness inspection.
# =====================================================================

FD002_FUSION_CANDIDATE_RANKING_628HC3A1 = (
    FD002_FUSION_CANDIDATE_SUMMARY_628HC3A1
    .sort_values(
        by=[
            "median_gain_80",
            "median_gain_70",
            "median_gain_90",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- CANDIDATE FUSION RANKING ---"
)

display(
    FD002_FUSION_CANDIDATE_RANKING_628HC3A1[
        [
            "candidate",
            "is_combined_candidate",
            "clean_gain_80",
            "median_scenario_spearman",
            "median_gain_90",
            "median_gain_80",
            "median_gain_70",
            "mean_gain_80",
            "q10_gain_80",
            "positive_gain_80",
            "zero_gain_80",
            "negative_gain_80",
            "median_stuck_gain_80",
            "median_non_stuck_gain_80",
            "delta_median_gain80_vs_uncertainty",
        ]
    ]
)


# =====================================================================
# 16. FAMILY x SEVERITY RESPONSE
# =====================================================================

family_severity_628hc3a1 = (
    scenario_results_628hc3a1
    .groupby(
        [
            "candidate",
            "family",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),
        median_full_mae=(
            "full_mae",
            "median",
        ),
        median_spearman=(
            "risk_error_spearman",
            "median",
        ),
        median_gain_90=(
            "gain_at_90pct_coverage",
            "median",
        ),
        median_gain_80=(
            "gain_at_80pct_coverage",
            "median",
        ),
        median_gain_70=(
            "gain_at_70pct_coverage",
            "median",
        ),
        mean_gain_80=(
            "gain_at_80pct_coverage",
            "mean",
        ),
        median_risk=(
            "risk_median",
            "median",
        ),
        median_largest_tie_fraction=(
            "largest_tie_fraction",
            "median",
        ),
    )
)

FD002_FUSION_FAMILY_SEVERITY_RESPONSE_628HC3A1 = (
    family_severity_628hc3a1
)


# Show the three highest-ranked COMBINED candidates
top_combined_candidates_628hc3a1 = (
    FD002_FUSION_CANDIDATE_RANKING_628HC3A1[
        FD002_FUSION_CANDIDATE_RANKING_628HC3A1[
            "is_combined_candidate"
        ]
    ]
    .head(3)[
        "candidate"
    ]
    .tolist()
)

print(
    "\n--- FAMILY x SEVERITY RESPONSE "
    "FOR TOP 3 COMBINED CANDIDATES ---"
)

display(
    FD002_FUSION_FAMILY_SEVERITY_RESPONSE_628HC3A1[
        FD002_FUSION_FAMILY_SEVERITY_RESPONSE_628HC3A1[
            "candidate"
        ].isin(
            top_combined_candidates_628hc3a1
        )
    ]
    .sort_values(
        [
            "candidate",
            "family",
            "severity",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =====================================================================
# 17. BEST / WORST SCENARIOS FOR TOP-RANKED COMBINED CANDIDATE
#
# Diagnostic only.
# The candidate is NOT frozen here.
# =====================================================================

if len(
    top_combined_candidates_628hc3a1
) > 0:

    diagnostic_candidate_628hc3a1 = (
        top_combined_candidates_628hc3a1[
            0
        ]
    )

    diagnostic_rows_628hc3a1 = (
        scenario_results_628hc3a1[
            scenario_results_628hc3a1[
                "candidate"
            ] == diagnostic_candidate_628hc3a1
        ]
        .copy()
    )

    print(
        "\n--- DIAGNOSTIC TOP-RANKED COMBINED CANDIDATE ---"
    )

    print(
        "Candidate:",
        diagnostic_candidate_628hc3a1
    )

    print(
        "\n--- BEST 15 SCENARIOS AT 80% COVERAGE ---"
    )

    display(
        diagnostic_rows_628hc3a1
        .sort_values(
            "gain_at_80pct_coverage",
            ascending=False,
        )
        .head(15)[
            [
                "scenario_id",
                "family",
                "sensor",
                "severity",
                "full_mae",
                "risk_error_spearman",
                "mae_at_80pct_coverage",
                "gain_at_80pct_coverage",
                "risk_median",
                "largest_tie_fraction",
            ]
        ]
    )

    print(
        "\n--- WORST 15 SCENARIOS AT 80% COVERAGE ---"
    )

    display(
        diagnostic_rows_628hc3a1
        .sort_values(
            "gain_at_80pct_coverage",
            ascending=True,
        )
        .head(15)[
            [
                "scenario_id",
                "family",
                "sensor",
                "severity",
                "full_mae",
                "risk_error_spearman",
                "mae_at_80pct_coverage",
                "gain_at_80pct_coverage",
                "risk_median",
                "largest_tie_fraction",
            ]
        ]
    )

else:

    diagnostic_candidate_628hc3a1 = None


# =====================================================================
# 18. FINAL AUDIT
# =====================================================================

expected_result_rows_628hc3a1 = (
    len(
        FUSION_CANDIDATES_628HC3A1
    )
    * n_scenarios_628hc3a1
)

audit_rows_628hc3a1 = [
    {
        "check": "scenario_count_315",
        "passed": bool(
            n_scenarios_628hc3a1 == 315
        ),
    },
    {
        "check": "prediction_matrix_shape",
        "passed": bool(
            pred_matrix_628hc3a1.shape
            ==
            expected_matrix_shape_628hc3a1
        ),
    },
    {
        "check": "uncertainty_matrix_shape",
        "passed": bool(
            u_matrix_628hc3a1.shape
            ==
            expected_matrix_shape_628hc3a1
        ),
    },
    {
        "check": "quality_matrix_shape",
        "passed": bool(
            q_matrix_628hc3a1.shape
            ==
            expected_matrix_shape_628hc3a1
        ),
    },
    {
        "check": "clean_uncertainty_exact",
        "passed": bool(
            clean_u_exact_628hc3a1
        ),
    },
    {
        "check": "candidate_result_row_count",
        "passed": bool(
            len(
                FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1
            )
            ==
            expected_result_rows_628hc3a1
        ),
    },
    {
        "check": "scenario_metrics_finite",
        "passed": bool(
            np.all(
                np.isfinite(
                    FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1[
                        [
                            "full_mae",
                            "mae_at_90pct_coverage",
                            "mae_at_80pct_coverage",
                            "mae_at_70pct_coverage",
                            "gain_at_90pct_coverage",
                            "gain_at_80pct_coverage",
                            "gain_at_70pct_coverage",
                            "risk_median",
                            "risk_mean",
                            "largest_tie_fraction",
                        ]
                    ].to_numpy(
                        dtype=np.float64
                    )
                )
            )
        ),
    },
]

FD002_FUSION_CANDIDATE_AUDIT_628HC3A1 = (
    pd.DataFrame(
        audit_rows_628hc3a1
    )
)

print(
    "\n--- FINAL C3-A1 AUDIT ---"
)

display(
    FD002_FUSION_CANDIDATE_AUDIT_628HC3A1
)

all_checks_passed_628hc3a1 = bool(
    FD002_FUSION_CANDIDATE_AUDIT_628HC3A1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a1:
    raise RuntimeError(
        "One or more C3-A1 audit checks failed."
    )


# =====================================================================
# 19. FROZEN STEP CONTRACT
#
# These objects document what C3-A1 did and DID NOT do.
# =====================================================================

FD002_FUSION_CANDIDATE_CONTRACT_628HC3A1 = {
    "uncertainty_source":
        "uncertainty_risk_matrix_628hc3a0r1",

    "clean_uncertainty_source":
        "clean_uncertainty_risk_628hc3a0r1",

    "sensor_quality_source":
        "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",

    "clean_sensor_quality_source":
        "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B"
        "['sensor_quality_risk']",

    "quality_fit_q95_marker":
        float(
            quality_q95_marker_628hc3a1
        ),

    "coverages":
        tuple(
            COVERAGES_628HC3A1
        ),

    "tie_policy":
        "expected absolute error within exact tied-risk boundary group",

    "candidate_rules":
        tuple(
            FUSION_CANDIDATES_628HC3A1
        ),

    "primary_development_diagnostic":
        "median MAE gain at 80% coverage across 315 DEV corruption scenarios",

    "fusion_rule_selected":
        False,

    "combination_weight_frozen":
        False,

    "abstention_threshold_selected":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "model_executed":
        False,

    "model_retrained":
        False,

    "files_written":
        False,

    "all_audit_checks_passed":
        all_checks_passed_628hc3a1,
}


# =====================================================================
# 20. FINAL SUMMARY
# =====================================================================

print("\n" + "=" * 94)
print("STEP 6.28H-C3-A1 FINAL SUMMARY")
print("=" * 94)

print(
    "Scenarios evaluated             :",
    n_scenarios_628hc3a1
)

print(
    "DEV windows/scenario            :",
    n_dev_628hc3a1
)

print(
    "Fusion candidates evaluated     :",
    len(
        FUSION_CANDIDATES_628HC3A1
    )
)

print(
    "Scenario-candidate evaluations  :",
    len(
        FD002_FUSION_CANDIDATE_SCENARIO_RESULTS_628HC3A1
    )
)

print(
    "Frozen uncertainty chain exact  :",
    clean_u_exact_628hc3a1
)

print(
    "FIT sensor-quality q95 marker   :",
    quality_q95_marker_628hc3a1
)

print(
    "Top-ranked diagnostic candidate :",
    diagnostic_candidate_628hc3a1
)

print(
    "All C3-A1 checks passed         :",
    all_checks_passed_628hc3a1
)

print(
    "Fusion rule selected            : False"
)

print(
    "Combination weight frozen       : False"
)

print(
    "Abstention threshold selected   : False"
)

print(
    "DEV RUL labels used             : True"
)

print(
    "Official TEST labels read       : False"
)

print(
    "TEST scored                     : False"
)

print(
    "Model executed                  : False"
)

print(
    "Model retrained                 : False"
)

print(
    "Files written                   : False"
)

print(
    "Runtime (sec)                   :",
    total_runtime_628hc3a1
)

print("\n=== END STEP 6.28H-C3-A1 ===")

In [ ]:
# =====================================================================
# STEP 6.28H-C3-A2
# ROBUST COMBINED-RELIABILITY FUSION STABILITY EVALUATION
# =====================================================================
#
# PURPOSE
# -------
# 1. Evaluate a finer, predefined gamma grid for the gated
#    uncertainty + sensor-quality fusion rule.
#
# 2. Use ONLY frozen FD002 DEV corruption results.
#
# 3. Perform leave-one-corruption-family-out (LOFO) evaluation:
#       bias
#       noise
#       drift
#       dropout
#       stuck
#
# 4. Examine robustness across:
#       90%, 80%, 70% coverage
#
# 5. Do NOT:
#       - read official TEST labels
#       - score TEST
#       - retrain the model
#       - select an abstention threshold
#       - write files
#       - freeze the final fusion rule
#
# IMPORTANT
# ---------
# This is a DEVELOPMENT-ONLY robustness/stability experiment.
# =====================================================================

import numpy as np
import pandas as pd
import time

print("=" * 86)
print("STEP 6.28H-C3-A2 — ROBUST COMBINED-RELIABILITY FUSION STABILITY EVALUATION")
print("=" * 86)


# =====================================================================
# 1. REQUIRED OBJECTS
# =====================================================================

required_objects_628hc3a2 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTED_DEV_PREDICTIONS_628HB",
    "Y_DEV_PRIMARY_FD002_628D",
    "DEV_PREDICTIONS_FD002_628EB",
    "FD002_DEV_SELECTIVE_RISK_628FCR2",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B",
]

print("\n--- REQUIRED CORE OBJECTS ---")

missing_628hc3a2 = []

for name_628hc3a2 in required_objects_628hc3a2:
    exists_628hc3a2 = name_628hc3a2 in globals()
    print(f"{name_628hc3a2:<52} : {exists_628hc3a2}")

    if not exists_628hc3a2:
        missing_628hc3a2.append(name_628hc3a2)

# Frozen corrupted uncertainty matrix:
# Prefer the exact C3-A0-R1 recovered object.
if "uncertainty_risk_matrix_628hc3a0r1" in globals():
    UNCERTAINTY_MATRIX_SOURCE_628HC3A2 = (
        "uncertainty_risk_matrix_628hc3a0r1"
    )
    uncertainty_matrix_628hc3a2 = np.asarray(
        globals()["uncertainty_risk_matrix_628hc3a0r1"],
        dtype=np.float64
    )

elif "FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D" in globals():
    UNCERTAINTY_MATRIX_SOURCE_628HC3A2 = (
        "FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D"
    )
    uncertainty_matrix_628hc3a2 = np.asarray(
        globals()["FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC1D"],
        dtype=np.float64
    )

else:
    UNCERTAINTY_MATRIX_SOURCE_628HC3A2 = None
    uncertainty_matrix_628hc3a2 = None
    missing_628hc3a2.append(
        "frozen corrupted uncertainty-risk matrix"
    )

print(
    f"{'Frozen corrupted uncertainty-risk matrix':<52} : "
    f"{uncertainty_matrix_628hc3a2 is not None}"
)

if missing_628hc3a2:
    raise RuntimeError(
        "STEP 6.28H-C3-A2 cannot run because required objects "
        f"are missing: {missing_628hc3a2}"
    )


# =====================================================================
# 2. LOAD FROZEN OBJECTS
# =====================================================================

manifest_628hc3a2 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

prediction_matrix_628hc3a2 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64
)

quality_matrix_628hc3a2 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

y_dev_628hc3a2 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64
).reshape(-1)

clean_prediction_628hc3a2 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64
).reshape(-1)

clean_uncertainty_628hc3a2 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64
).reshape(-1)


# ---------------------------------------------------------------------
# Recover clean calibrated sensor-quality risk correctly from dictionary.
# ---------------------------------------------------------------------

clean_quality_container_628hc3a2 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

if not isinstance(clean_quality_container_628hc3a2, dict):
    raise TypeError(
        "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B was expected "
        "to be a dictionary."
    )

if "sensor_quality_risk" not in clean_quality_container_628hc3a2:
    raise KeyError(
        "Dictionary CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
        "does not contain key 'sensor_quality_risk'."
    )

clean_quality_628hc3a2 = np.asarray(
    clean_quality_container_628hc3a2["sensor_quality_risk"],
    dtype=np.float64
).reshape(-1)

quality_fit_q95_628hc3a2 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)


# =====================================================================
# 3. POPULATION CONTRACT
# =====================================================================

n_scenarios_628hc3a2 = len(manifest_628hc3a2)
n_dev_628hc3a2 = len(y_dev_628hc3a2)

expected_matrix_shape_628hc3a2 = (
    n_scenarios_628hc3a2,
    n_dev_628hc3a2
)

print("\n--- POPULATION CONTRACT ---")
print("Scenarios                    :", n_scenarios_628hc3a2)
print("DEV windows                  :", n_dev_628hc3a2)
print("Prediction matrix            :", prediction_matrix_628hc3a2.shape)
print("Uncertainty-risk matrix      :", uncertainty_matrix_628hc3a2.shape)
print("Sensor-quality matrix        :", quality_matrix_628hc3a2.shape)
print("Clean prediction             :", clean_prediction_628hc3a2.shape)
print("Clean uncertainty risk       :", clean_uncertainty_628hc3a2.shape)
print("Clean sensor-quality risk    :", clean_quality_628hc3a2.shape)
print("Quality FIT q95 marker       :", quality_fit_q95_628hc3a2)
print("Uncertainty matrix source    :", UNCERTAINTY_MATRIX_SOURCE_628HC3A2)

if n_scenarios_628hc3a2 != 315:
    raise RuntimeError(
        f"Expected 315 frozen scenarios, got {n_scenarios_628hc3a2}."
    )

if prediction_matrix_628hc3a2.shape != expected_matrix_shape_628hc3a2:
    raise RuntimeError(
        "Corrupted prediction matrix has unexpected shape."
    )

if uncertainty_matrix_628hc3a2.shape != expected_matrix_shape_628hc3a2:
    raise RuntimeError(
        "Frozen corrupted uncertainty-risk matrix has unexpected shape."
    )

if quality_matrix_628hc3a2.shape != expected_matrix_shape_628hc3a2:
    raise RuntimeError(
        "Frozen corrupted sensor-quality matrix has unexpected shape."
    )

for vector_name_628hc3a2, vector_628hc3a2 in [
    ("clean_prediction", clean_prediction_628hc3a2),
    ("clean_uncertainty", clean_uncertainty_628hc3a2),
    ("clean_quality", clean_quality_628hc3a2),
]:
    if vector_628hc3a2.shape != (n_dev_628hc3a2,):
        raise RuntimeError(
            f"{vector_name_628hc3a2} has unexpected shape "
            f"{vector_628hc3a2.shape}."
        )


# =====================================================================
# 4. MANIFEST ORDER CONTRACT
# =====================================================================

required_manifest_cols_628hc3a2 = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
]

missing_manifest_cols_628hc3a2 = [
    col_628hc3a2
    for col_628hc3a2 in required_manifest_cols_628hc3a2
    if col_628hc3a2 not in manifest_628hc3a2.columns
]

if missing_manifest_cols_628hc3a2:
    raise RuntimeError(
        "Manifest is missing required columns: "
        f"{missing_manifest_cols_628hc3a2}"
    )

expected_numbers_628hc3a2 = np.arange(1, 316)

scenario_numbers_exact_628hc3a2 = np.array_equal(
    manifest_628hc3a2["scenario_number"]
    .to_numpy(dtype=int),
    expected_numbers_628hc3a2
)

scenario_ids_expected_628hc3a2 = np.array(
    [
        f"FD002_SCENARIO_{i:03d}"
        for i in expected_numbers_628hc3a2
    ],
    dtype=object
)

scenario_ids_exact_628hc3a2 = np.array_equal(
    manifest_628hc3a2["scenario_id"]
    .astype(str)
    .to_numpy(),
    scenario_ids_expected_628hc3a2
)

print("\n--- MANIFEST ORDER CONTRACT ---")
print(
    "Scenario numbers 001–315 exact :",
    scenario_numbers_exact_628hc3a2
)
print(
    "Scenario IDs exact              :",
    scenario_ids_exact_628hc3a2
)

if not scenario_numbers_exact_628hc3a2:
    raise RuntimeError(
        "Manifest scenario-number order is not exact."
    )

if not scenario_ids_exact_628hc3a2:
    raise RuntimeError(
        "Manifest scenario-ID order is not exact."
    )


# =====================================================================
# 5. NUMERICAL INTEGRITY
# =====================================================================

numerical_checks_628hc3a2 = {
    "prediction_matrix_finite":
        bool(np.isfinite(prediction_matrix_628hc3a2).all()),

    "uncertainty_matrix_finite":
        bool(np.isfinite(uncertainty_matrix_628hc3a2).all()),

    "quality_matrix_finite":
        bool(np.isfinite(quality_matrix_628hc3a2).all()),

    "y_dev_finite":
        bool(np.isfinite(y_dev_628hc3a2).all()),

    "clean_prediction_finite":
        bool(np.isfinite(clean_prediction_628hc3a2).all()),

    "clean_uncertainty_finite":
        bool(np.isfinite(clean_uncertainty_628hc3a2).all()),

    "clean_quality_finite":
        bool(np.isfinite(clean_quality_628hc3a2).all()),

    "uncertainty_matrix_within_0_1":
        bool(
            np.all(
                (uncertainty_matrix_628hc3a2 >= 0.0)
                &
                (uncertainty_matrix_628hc3a2 <= 1.0)
            )
        ),

    "quality_matrix_within_0_1":
        bool(
            np.all(
                (quality_matrix_628hc3a2 >= 0.0)
                &
                (quality_matrix_628hc3a2 <= 1.0)
            )
        ),

    "clean_uncertainty_within_0_1":
        bool(
            np.all(
                (clean_uncertainty_628hc3a2 >= 0.0)
                &
                (clean_uncertainty_628hc3a2 <= 1.0)
            )
        ),

    "clean_quality_within_0_1":
        bool(
            np.all(
                (clean_quality_628hc3a2 >= 0.0)
                &
                (clean_quality_628hc3a2 <= 1.0)
            )
        ),
}

print("\n--- NUMERICAL INTEGRITY ---")

for check_name_628hc3a2, passed_628hc3a2 in (
    numerical_checks_628hc3a2.items()
):
    print(
        f"{check_name_628hc3a2:<36} : {passed_628hc3a2}"
    )

if not all(numerical_checks_628hc3a2.values()):
    raise RuntimeError(
        "Numerical-integrity audit failed."
    )


# =====================================================================
# 6. FUSION DEFINITION
# =====================================================================
#
# Sensor-quality risk is used ONLY when it exceeds the FIT-only q95
# diagnostic reference.
#
# Normalized excess:
#
#     E(Q) = clip(
#               (Q - q95_FIT) / (1 - q95_FIT),
#               0,
#               1
#            )
#
# Combined risk:
#
#     R_gamma = U + (1 - U) * gamma * E(Q)
#
# gamma = 0  -> uncertainty-only baseline.
#
# No DEV labels are used to CREATE the risk values.
# DEV labels are used only to EVALUATE candidate ranking quality.
# =====================================================================

def quality_excess_628hc3a2(
    quality_values,
    q95_marker
):
    quality_values = np.asarray(
        quality_values,
        dtype=np.float64
    )

    denominator = 1.0 - float(q95_marker)

    if denominator <= 0.0:
        raise ValueError(
            "FIT q95 marker must be strictly below 1."
        )

    excess = (
        quality_values - float(q95_marker)
    ) / denominator

    return np.clip(excess, 0.0, 1.0)


def fuse_risk_628hc3a2(
    uncertainty_risk,
    quality_risk,
    gamma,
    q95_marker
):
    uncertainty_risk = np.asarray(
        uncertainty_risk,
        dtype=np.float64
    )

    quality_risk = np.asarray(
        quality_risk,
        dtype=np.float64
    )

    gamma = float(gamma)

    if gamma < 0.0 or gamma > 1.0:
        raise ValueError(
            "gamma must lie within [0, 1]."
        )

    if uncertainty_risk.shape != quality_risk.shape:
        raise ValueError(
            "Uncertainty and sensor-quality arrays must have "
            "identical shapes."
        )

    excess = quality_excess_628hc3a2(
        quality_risk,
        q95_marker
    )

    combined = (
        uncertainty_risk
        +
        (1.0 - uncertainty_risk)
        * gamma
        * excess
    )

    return np.clip(combined, 0.0, 1.0)


# =====================================================================
# 7. TIE-AWARE SELECTIVE MAE
# =====================================================================
#
# Lower risk is accepted first.
#
# If the requested coverage boundary falls inside an exact tied-risk
# group, expected error is used for the fractional part of that group.
#
# This avoids arbitrary index-order decisions inside tied risks.
# =====================================================================

def selective_mae_tie_aware_628hc3a2(
    risk,
    absolute_error,
    coverage
):
    risk = np.asarray(
        risk,
        dtype=np.float64
    ).reshape(-1)

    absolute_error = np.asarray(
        absolute_error,
        dtype=np.float64
    ).reshape(-1)

    if risk.shape != absolute_error.shape:
        raise ValueError(
            "risk and absolute_error must have identical shapes."
        )

    if not np.isfinite(risk).all():
        raise ValueError("risk contains non-finite values.")

    if not np.isfinite(absolute_error).all():
        raise ValueError(
            "absolute_error contains non-finite values."
        )

    coverage = float(coverage)

    if coverage <= 0.0 or coverage > 1.0:
        raise ValueError(
            "coverage must lie in (0, 1]."
        )

    n = len(risk)

    if n == 0:
        raise ValueError("Empty risk array.")

    target_count = coverage * n

    order = np.argsort(
        risk,
        kind="mergesort"
    )

    sorted_risk = risk[order]
    sorted_error = absolute_error[order]

    # Boundary observation.
    boundary_position = min(
        int(np.ceil(target_count)) - 1,
        n - 1
    )

    boundary_risk = sorted_risk[
        boundary_position
    ]

    lower_end = np.searchsorted(
        sorted_risk,
        boundary_risk,
        side="left"
    )

    upper_end = np.searchsorted(
        sorted_risk,
        boundary_risk,
        side="right"
    )

    lower_count = float(lower_end)
    tied_count = float(
        upper_end - lower_end
    )

    required_from_tie = (
        target_count - lower_count
    )

    if required_from_tie < -1e-12:
        raise RuntimeError(
            "Tie-aware coverage calculation failed."
        )

    required_from_tie = max(
        0.0,
        required_from_tie
    )

    if tied_count <= 0.0:
        raise RuntimeError(
            "Boundary tied-risk group is empty."
        )

    tie_fraction = (
        required_from_tie
        / tied_count
    )

    tie_fraction = float(
        np.clip(
            tie_fraction,
            0.0,
            1.0
        )
    )

    lower_error_sum = float(
        sorted_error[:lower_end].sum()
    )

    tied_error_sum = float(
        sorted_error[
            lower_end:upper_end
        ].sum()
    )

    expected_accepted_error = (
        lower_error_sum
        +
        tie_fraction
        * tied_error_sum
    )

    mae = (
        expected_accepted_error
        / target_count
    )

    largest_tie_fraction = (
        tied_count / n
    )

    return {
        "mae": float(mae),
        "boundary_risk": float(boundary_risk),
        "boundary_group_fraction_of_population":
            float(largest_tie_fraction),
        "boundary_group_acceptance_fraction":
            float(tie_fraction),
    }


# =====================================================================
# 8. PREDEFINED GAMMA GRID
# =====================================================================
#
# IMPORTANT:
# This grid is defined before inspecting C3-A2 results.
#
# gamma = 0.00 is the frozen uncertainty-only reference.
# =====================================================================

GAMMA_GRID_628HC3A2 = np.round(
    np.arange(
        0.00,
        1.0001,
        0.05
    ),
    2
)

COVERAGES_628HC3A2 = (
    0.90,
    0.80,
    0.70
)

CORRUPTION_FAMILIES_628HC3A2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

print("\n--- PREDEFINED ROBUSTNESS GRID ---")
print("Gamma grid :", GAMMA_GRID_628HC3A2.tolist())
print("Gamma count:", len(GAMMA_GRID_628HC3A2))
print("Coverages  :", COVERAGES_628HC3A2)
print("Families   :", CORRUPTION_FAMILIES_628HC3A2)


# =====================================================================
# 9. CLEAN DEV BASELINE
# =====================================================================

clean_abs_error_628hc3a2 = np.abs(
    clean_prediction_628hc3a2
    -
    y_dev_628hc3a2
)

clean_full_mae_628hc3a2 = float(
    np.mean(
        clean_abs_error_628hc3a2
    )
)

clean_gamma_rows_628hc3a2 = []

for gamma_628hc3a2 in GAMMA_GRID_628HC3A2:

    clean_combined_risk_628hc3a2 = (
        fuse_risk_628hc3a2(
            clean_uncertainty_628hc3a2,
            clean_quality_628hc3a2,
            gamma_628hc3a2,
            quality_fit_q95_628hc3a2
        )
    )

    clean_row_628hc3a2 = {
        "gamma": float(gamma_628hc3a2),
        "clean_full_mae":
            clean_full_mae_628hc3a2,
        "risk_median":
            float(
                np.median(
                    clean_combined_risk_628hc3a2
                )
            ),
        "risk_mean":
            float(
                np.mean(
                    clean_combined_risk_628hc3a2
                )
            ),
        "unique_risk_values":
            int(
                np.unique(
                    clean_combined_risk_628hc3a2
                ).size
            ),
    }

    for cov_628hc3a2 in COVERAGES_628HC3A2:

        eval_clean_628hc3a2 = (
            selective_mae_tie_aware_628hc3a2(
                clean_combined_risk_628hc3a2,
                clean_abs_error_628hc3a2,
                cov_628hc3a2
            )
        )

        pct_628hc3a2 = int(
            round(
                100 * cov_628hc3a2
            )
        )

        clean_row_628hc3a2[
            f"mae_{pct_628hc3a2}"
        ] = eval_clean_628hc3a2["mae"]

        clean_row_628hc3a2[
            f"gain_{pct_628hc3a2}"
        ] = (
            clean_full_mae_628hc3a2
            -
            eval_clean_628hc3a2["mae"]
        )

    clean_gamma_rows_628hc3a2.append(
        clean_row_628hc3a2
    )

FD002_C3A2_CLEAN_GAMMA_RESPONSE_628HC3A2 = (
    pd.DataFrame(
        clean_gamma_rows_628hc3a2
    )
)

print("\n--- CLEAN DEV GAMMA RESPONSE ---")
print(
    "Clean full MAE:",
    clean_full_mae_628hc3a2
)

display(
    FD002_C3A2_CLEAN_GAMMA_RESPONSE_628HC3A2[
        [
            "gamma",
            "gain_90",
            "gain_80",
            "gain_70",
            "risk_median",
            "unique_risk_values",
        ]
    ]
)


# =====================================================================
# 10. RUN ALL GAMMAS OVER ALL 315 CORRUPTED DEV SCENARIOS
# =====================================================================

scenario_rows_628hc3a2 = []

total_start_628hc3a2 = time.perf_counter()

total_evaluations_628hc3a2 = (
    len(GAMMA_GRID_628HC3A2)
    *
    n_scenarios_628hc3a2
)

completed_628hc3a2 = 0

print(
    "\n--- RUNNING ROBUST FUSION GRID OVER "
    "315 CORRUPTED DEV SCENARIOS ---"
)

for gamma_index_628hc3a2, gamma_628hc3a2 in enumerate(
    GAMMA_GRID_628HC3A2,
    start=1
):

    gamma_start_628hc3a2 = time.perf_counter()

    for scenario_index_628hc3a2 in range(
        n_scenarios_628hc3a2
    ):

        scenario_manifest_row_628hc3a2 = (
            manifest_628hc3a2.iloc[
                scenario_index_628hc3a2
            ]
        )

        pred_628hc3a2 = (
            prediction_matrix_628hc3a2[
                scenario_index_628hc3a2
            ]
        )

        u_628hc3a2 = (
            uncertainty_matrix_628hc3a2[
                scenario_index_628hc3a2
            ]
        )

        q_628hc3a2 = (
            quality_matrix_628hc3a2[
                scenario_index_628hc3a2
            ]
        )

        risk_628hc3a2 = (
            fuse_risk_628hc3a2(
                u_628hc3a2,
                q_628hc3a2,
                gamma_628hc3a2,
                quality_fit_q95_628hc3a2
            )
        )

        abs_error_628hc3a2 = np.abs(
            pred_628hc3a2
            -
            y_dev_628hc3a2
        )

        full_mae_628hc3a2 = float(
            np.mean(
                abs_error_628hc3a2
            )
        )

        result_row_628hc3a2 = {
            "gamma":
                float(gamma_628hc3a2),

            "scenario_id":
                str(
                    scenario_manifest_row_628hc3a2[
                        "scenario_id"
                    ]
                ),

            "scenario_number":
                int(
                    scenario_manifest_row_628hc3a2[
                        "scenario_number"
                    ]
                ),

            "family":
                str(
                    scenario_manifest_row_628hc3a2[
                        "family"
                    ]
                ),

            "sensor":
                str(
                    scenario_manifest_row_628hc3a2[
                        "sensor"
                    ]
                ),

            "severity":
                str(
                    scenario_manifest_row_628hc3a2[
                        "severity"
                    ]
                ),

            "full_mae":
                full_mae_628hc3a2,

            "risk_median":
                float(
                    np.median(
                        risk_628hc3a2
                    )
                ),

            "risk_mean":
                float(
                    np.mean(
                        risk_628hc3a2
                    )
                ),

            "unique_risk_values":
                int(
                    np.unique(
                        risk_628hc3a2
                    ).size
                ),
        }

        for cov_628hc3a2 in COVERAGES_628HC3A2:

            eval_result_628hc3a2 = (
                selective_mae_tie_aware_628hc3a2(
                    risk_628hc3a2,
                    abs_error_628hc3a2,
                    cov_628hc3a2
                )
            )

            pct_628hc3a2 = int(
                round(
                    100 * cov_628hc3a2
                )
            )

            mae_cov_628hc3a2 = (
                eval_result_628hc3a2[
                    "mae"
                ]
            )

            result_row_628hc3a2[
                f"mae_{pct_628hc3a2}"
            ] = mae_cov_628hc3a2

            result_row_628hc3a2[
                f"gain_{pct_628hc3a2}"
            ] = (
                full_mae_628hc3a2
                -
                mae_cov_628hc3a2
            )

            result_row_628hc3a2[
                f"boundary_tie_fraction_{pct_628hc3a2}"
            ] = (
                eval_result_628hc3a2[
                    "boundary_group_fraction_of_population"
                ]
            )

        scenario_rows_628hc3a2.append(
            result_row_628hc3a2
        )

        completed_628hc3a2 += 1

    gamma_elapsed_628hc3a2 = (
        time.perf_counter()
        -
        gamma_start_628hc3a2
    )

    total_elapsed_now_628hc3a2 = (
        time.perf_counter()
        -
        total_start_628hc3a2
    )

    print(
        f"[{gamma_index_628hc3a2:02d}/"
        f"{len(GAMMA_GRID_628HC3A2):02d}] "
        f"gamma={gamma_628hc3a2:.2f} "
        f"| elapsed_gamma={gamma_elapsed_628hc3a2:.2f}s "
        f"| total={total_elapsed_now_628hc3a2:.1f}s"
    )


FD002_C3A2_SCENARIO_METRICS_628HC3A2 = pd.DataFrame(
    scenario_rows_628hc3a2
)

runtime_628hc3a2 = (
    time.perf_counter()
    -
    total_start_628hc3a2
)

print("\n--- FULL ROBUSTNESS GRID CREATED ---")
print(
    "Scenario-gamma rows :",
    len(
        FD002_C3A2_SCENARIO_METRICS_628HC3A2
    )
)
print(
    "Expected rows       :",
    total_evaluations_628hc3a2
)
print(
    "Runtime (sec)       :",
    runtime_628hc3a2
)


# =====================================================================
# 11. GLOBAL GAMMA ROBUSTNESS SUMMARY
# =====================================================================

clean_gain_lookup_628hc3a2 = (
    FD002_C3A2_CLEAN_GAMMA_RESPONSE_628HC3A2
    .set_index("gamma")
)

global_rows_628hc3a2 = []

for gamma_628hc3a2 in GAMMA_GRID_628HC3A2:

    gamma_df_628hc3a2 = (
        FD002_C3A2_SCENARIO_METRICS_628HC3A2[
            FD002_C3A2_SCENARIO_METRICS_628HC3A2[
                "gamma"
            ]
            ==
            float(gamma_628hc3a2)
        ]
        .copy()
    )

    family_medians_628hc3a2 = (
        gamma_df_628hc3a2
        .groupby(
            "family",
            observed=True
        )["gain_80"]
        .median()
    )

    worst_family_median_gain80_628hc3a2 = float(
        family_medians_628hc3a2.min()
    )

    clean_gain80_628hc3a2 = float(
        clean_gain_lookup_628hc3a2.loc[
            float(gamma_628hc3a2),
            "gain_80"
        ]
    )

    robust_score_628hc3a2 = float(
        min(
            clean_gain80_628hc3a2,
            worst_family_median_gain80_628hc3a2
        )
    )

    global_rows_628hc3a2.append(
        {
            "gamma":
                float(gamma_628hc3a2),

            "clean_gain_90":
                float(
                    clean_gain_lookup_628hc3a2.loc[
                        float(gamma_628hc3a2),
                        "gain_90"
                    ]
                ),

            "clean_gain_80":
                clean_gain80_628hc3a2,

            "clean_gain_70":
                float(
                    clean_gain_lookup_628hc3a2.loc[
                        float(gamma_628hc3a2),
                        "gain_70"
                    ]
                ),

            "median_gain_90":
                float(
                    gamma_df_628hc3a2[
                        "gain_90"
                    ].median()
                ),

            "median_gain_80":
                float(
                    gamma_df_628hc3a2[
                        "gain_80"
                    ].median()
                ),

            "median_gain_70":
                float(
                    gamma_df_628hc3a2[
                        "gain_70"
                    ].median()
                ),

            "mean_gain_80":
                float(
                    gamma_df_628hc3a2[
                        "gain_80"
                    ].mean()
                ),

            "q10_gain_80":
                float(
                    gamma_df_628hc3a2[
                        "gain_80"
                    ].quantile(0.10)
                ),

            "positive_gain_80":
                int(
                    (
                        gamma_df_628hc3a2[
                            "gain_80"
                        ]
                        >
                        0.0
                    ).sum()
                ),

            "zero_gain_80":
                int(
                    np.isclose(
                        gamma_df_628hc3a2[
                            "gain_80"
                        ].to_numpy(),
                        0.0,
                        atol=1e-12,
                        rtol=0.0
                    ).sum()
                ),

            "negative_gain_80":
                int(
                    (
                        gamma_df_628hc3a2[
                            "gain_80"
                        ]
                        <
                        -1e-12
                    ).sum()
                ),

            "worst_family_median_gain_80":
                worst_family_median_gain80_628hc3a2,

            "robust_score_80":
                robust_score_628hc3a2,
        }
    )


FD002_C3A2_GLOBAL_GAMMA_SUMMARY_628HC3A2 = (
    pd.DataFrame(
        global_rows_628hc3a2
    )
)


# Robust ranking:
# 1. maximize worst-case clean/family score
# 2. maximize q10 gain
# 3. maximize median gain
# 4. maximize mean gain
# 5. prefer smaller gamma if otherwise tied
FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2 = (
    FD002_C3A2_GLOBAL_GAMMA_SUMMARY_628HC3A2
    .sort_values(
        by=[
            "robust_score_80",
            "q10_gain_80",
            "median_gain_80",
            "mean_gain_80",
            "gamma",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)


print("\n--- GLOBAL ROBUST GAMMA RANKING ---")

display(
    FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2
)


# =====================================================================
# 12. LEAVE-ONE-CORRUPTION-FAMILY-OUT (LOFO) SELECTION
# =====================================================================
#
# For each held-out corruption family:
#
#   1. Selection is performed using only the OTHER four families.
#
#   2. gamma is selected using a robust criterion:
#
#        robust_score =
#          min(
#              clean DEV gain @80,
#              worst median family gain @80
#              among the four selection families
#          )
#
#   3. q10 gain, median gain and smaller gamma are tie-breakers.
#
#   4. The held-out family is then evaluated using the selected gamma.
#
# This gives us a much stronger indication of whether gamma choice
# generalizes to unseen corruption families.
# =====================================================================

lofo_rows_628hc3a2 = []

for heldout_family_628hc3a2 in (
    CORRUPTION_FAMILIES_628HC3A2
):

    training_df_628hc3a2 = (
        FD002_C3A2_SCENARIO_METRICS_628HC3A2[
            FD002_C3A2_SCENARIO_METRICS_628HC3A2[
                "family"
            ]
            !=
            heldout_family_628hc3a2
        ]
        .copy()
    )

    heldout_df_628hc3a2 = (
        FD002_C3A2_SCENARIO_METRICS_628HC3A2[
            FD002_C3A2_SCENARIO_METRICS_628HC3A2[
                "family"
            ]
            ==
            heldout_family_628hc3a2
        ]
        .copy()
    )

    selection_rows_628hc3a2 = []

    for gamma_628hc3a2 in GAMMA_GRID_628HC3A2:

        train_gamma_df_628hc3a2 = (
            training_df_628hc3a2[
                training_df_628hc3a2[
                    "gamma"
                ]
                ==
                float(gamma_628hc3a2)
            ]
        )

        train_family_medians_628hc3a2 = (
            train_gamma_df_628hc3a2
            .groupby(
                "family",
                observed=True
            )["gain_80"]
            .median()
        )

        train_worst_family_gain_628hc3a2 = float(
            train_family_medians_628hc3a2.min()
        )

        train_clean_gain_628hc3a2 = float(
            clean_gain_lookup_628hc3a2.loc[
                float(gamma_628hc3a2),
                "gain_80"
            ]
        )

        selection_rows_628hc3a2.append(
            {
                "gamma":
                    float(gamma_628hc3a2),

                "train_robust_score_80":
                    float(
                        min(
                            train_clean_gain_628hc3a2,
                            train_worst_family_gain_628hc3a2
                        )
                    ),

                "train_clean_gain_80":
                    train_clean_gain_628hc3a2,

                "train_worst_family_median_gain_80":
                    train_worst_family_gain_628hc3a2,

                "train_median_gain_80":
                    float(
                        train_gamma_df_628hc3a2[
                            "gain_80"
                        ].median()
                    ),

                "train_mean_gain_80":
                    float(
                        train_gamma_df_628hc3a2[
                            "gain_80"
                        ].mean()
                    ),

                "train_q10_gain_80":
                    float(
                        train_gamma_df_628hc3a2[
                            "gain_80"
                        ].quantile(0.10)
                    ),
            }
        )

    selection_df_628hc3a2 = pd.DataFrame(
        selection_rows_628hc3a2
    )

    selection_df_628hc3a2 = (
        selection_df_628hc3a2
        .sort_values(
            by=[
                "train_robust_score_80",
                "train_q10_gain_80",
                "train_median_gain_80",
                "train_mean_gain_80",
                "gamma",
            ],
            ascending=[
                False,
                False,
                False,
                False,
                True,
            ],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )

    selected_gamma_628hc3a2 = float(
        selection_df_628hc3a2.iloc[0][
            "gamma"
        ]
    )

    selected_train_row_628hc3a2 = (
        selection_df_628hc3a2.iloc[0]
    )

    heldout_selected_628hc3a2 = (
        heldout_df_628hc3a2[
            heldout_df_628hc3a2[
                "gamma"
            ]
            ==
            selected_gamma_628hc3a2
        ]
    )

    heldout_uncertainty_only_628hc3a2 = (
        heldout_df_628hc3a2[
            heldout_df_628hc3a2[
                "gamma"
            ]
            ==
            0.0
        ]
    )

    holdout_median_gain80_628hc3a2 = float(
        heldout_selected_628hc3a2[
            "gain_80"
        ].median()
    )

    baseline_holdout_median_gain80_628hc3a2 = float(
        heldout_uncertainty_only_628hc3a2[
            "gain_80"
        ].median()
    )

    lofo_rows_628hc3a2.append(
        {
            "heldout_family":
                heldout_family_628hc3a2,

            "selected_gamma":
                selected_gamma_628hc3a2,

            "train_robust_score_80":
                float(
                    selected_train_row_628hc3a2[
                        "train_robust_score_80"
                    ]
                ),

            "train_worst_family_median_gain_80":
                float(
                    selected_train_row_628hc3a2[
                        "train_worst_family_median_gain_80"
                    ]
                ),

            "train_median_gain_80":
                float(
                    selected_train_row_628hc3a2[
                        "train_median_gain_80"
                    ]
                ),

            "train_q10_gain_80":
                float(
                    selected_train_row_628hc3a2[
                        "train_q10_gain_80"
                    ]
                ),

            "clean_gain_80_at_selected_gamma":
                float(
                    selected_train_row_628hc3a2[
                        "train_clean_gain_80"
                    ]
                ),

            "heldout_scenarios":
                int(
                    len(
                        heldout_selected_628hc3a2
                    )
                ),

            "heldout_median_gain_90":
                float(
                    heldout_selected_628hc3a2[
                        "gain_90"
                    ].median()
                ),

            "heldout_median_gain_80":
                holdout_median_gain80_628hc3a2,

            "heldout_median_gain_70":
                float(
                    heldout_selected_628hc3a2[
                        "gain_70"
                    ].median()
                ),

            "heldout_mean_gain_80":
                float(
                    heldout_selected_628hc3a2[
                        "gain_80"
                    ].mean()
                ),

            "heldout_q10_gain_80":
                float(
                    heldout_selected_628hc3a2[
                        "gain_80"
                    ].quantile(0.10)
                ),

            "heldout_positive_gain_fraction_80":
                float(
                    np.mean(
                        heldout_selected_628hc3a2[
                            "gain_80"
                        ]
                        >
                        0.0
                    )
                ),

            "uncertainty_only_heldout_median_gain_80":
                baseline_holdout_median_gain80_628hc3a2,

            "delta_heldout_median_gain80_vs_uncertainty":
                float(
                    holdout_median_gain80_628hc3a2
                    -
                    baseline_holdout_median_gain80_628hc3a2
                ),
        }
    )


FD002_C3A2_LOFO_SUMMARY_628HC3A2 = pd.DataFrame(
    lofo_rows_628hc3a2
)

print(
    "\n--- LEAVE-ONE-CORRUPTION-FAMILY-OUT RESULTS ---"
)

display(
    FD002_C3A2_LOFO_SUMMARY_628HC3A2
)


# =====================================================================
# 13. LOFO GAMMA STABILITY
# =====================================================================

selected_gammas_628hc3a2 = (
    FD002_C3A2_LOFO_SUMMARY_628HC3A2[
        "selected_gamma"
    ]
    .to_numpy(dtype=np.float64)
)

gamma_stability_628hc3a2 = pd.DataFrame(
    {
        "metric": [
            "minimum_selected_gamma",
            "q25_selected_gamma",
            "median_selected_gamma",
            "q75_selected_gamma",
            "maximum_selected_gamma",
            "unique_selected_gammas",
        ],
        "value": [
            float(
                np.min(
                    selected_gammas_628hc3a2
                )
            ),
            float(
                np.quantile(
                    selected_gammas_628hc3a2,
                    0.25
                )
            ),
            float(
                np.median(
                    selected_gammas_628hc3a2
                )
            ),
            float(
                np.quantile(
                    selected_gammas_628hc3a2,
                    0.75
                )
            ),
            float(
                np.max(
                    selected_gammas_628hc3a2
                )
            ),
            int(
                np.unique(
                    selected_gammas_628hc3a2
                ).size
            ),
        ],
    }
)

FD002_C3A2_GAMMA_STABILITY_628HC3A2 = (
    gamma_stability_628hc3a2
)

print("\n--- LOFO GAMMA STABILITY ---")
display(
    FD002_C3A2_GAMMA_STABILITY_628HC3A2
)


# =====================================================================
# 14. FULL-DEV DIAGNOSTIC TOP GAMMA
# =====================================================================
#
# IMPORTANT:
# This is only a diagnostic recommendation.
# It is NOT frozen here.
# =====================================================================

diagnostic_gamma_628hc3a2 = float(
    FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2
    .iloc[0]["gamma"]
)

diagnostic_gamma_row_628hc3a2 = (
    FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2
    .iloc[0]
)

print(
    "\n--- TOP FULL-DEV ROBUSTNESS DIAGNOSTIC GAMMA ---"
)
print(
    "Diagnostic gamma:",
    diagnostic_gamma_628hc3a2
)
print(
    "Robust score @80:",
    diagnostic_gamma_row_628hc3a2[
        "robust_score_80"
    ]
)
print(
    "Median gain @80:",
    diagnostic_gamma_row_628hc3a2[
        "median_gain_80"
    ]
)
print(
    "Mean gain @80:",
    diagnostic_gamma_row_628hc3a2[
        "mean_gain_80"
    ]
)
print(
    "q10 gain @80:",
    diagnostic_gamma_row_628hc3a2[
        "q10_gain_80"
    ]
)
print(
    "Positive gain scenarios:",
    diagnostic_gamma_row_628hc3a2[
        "positive_gain_80"
    ],
    "/ 315"
)


# =====================================================================
# 15. FAMILY × SEVERITY RESPONSE FOR DIAGNOSTIC GAMMA
# =====================================================================

diagnostic_df_628hc3a2 = (
    FD002_C3A2_SCENARIO_METRICS_628HC3A2[
        FD002_C3A2_SCENARIO_METRICS_628HC3A2[
            "gamma"
        ]
        ==
        diagnostic_gamma_628hc3a2
    ]
    .copy()
)

family_severity_rows_628hc3a2 = []

for (
    family_628hc3a2,
    severity_628hc3a2
), group_628hc3a2 in diagnostic_df_628hc3a2.groupby(
    [
        "family",
        "severity",
    ],
    observed=True,
    sort=False
):

    family_severity_rows_628hc3a2.append(
        {
            "family":
                family_628hc3a2,

            "severity":
                severity_628hc3a2,

            "scenarios":
                int(
                    len(
                        group_628hc3a2
                    )
                ),

            "median_full_mae":
                float(
                    group_628hc3a2[
                        "full_mae"
                    ].median()
                ),

            "median_gain_90":
                float(
                    group_628hc3a2[
                        "gain_90"
                    ].median()
                ),

            "median_gain_80":
                float(
                    group_628hc3a2[
                        "gain_80"
                    ].median()
                ),

            "median_gain_70":
                float(
                    group_628hc3a2[
                        "gain_70"
                    ].median()
                ),

            "mean_gain_80":
                float(
                    group_628hc3a2[
                        "gain_80"
                    ].mean()
                ),

            "q10_gain_80":
                float(
                    group_628hc3a2[
                        "gain_80"
                    ].quantile(0.10)
                ),

            "positive_gain_80":
                int(
                    (
                        group_628hc3a2[
                            "gain_80"
                        ]
                        >
                        0.0
                    ).sum()
                ),
        }
    )


FD002_C3A2_FAMILY_SEVERITY_628HC3A2 = (
    pd.DataFrame(
        family_severity_rows_628hc3a2
    )
)

print(
    "\n--- FAMILY × SEVERITY RESPONSE "
    "FOR DIAGNOSTIC GAMMA ---"
)

display(
    FD002_C3A2_FAMILY_SEVERITY_628HC3A2
)


# =====================================================================
# 16. COMPARE DIAGNOSTIC GAMMA WITH UNCERTAINTY-ONLY
# =====================================================================

uncertainty_only_df_628hc3a2 = (
    FD002_C3A2_SCENARIO_METRICS_628HC3A2[
        FD002_C3A2_SCENARIO_METRICS_628HC3A2[
            "gamma"
        ]
        ==
        0.0
    ]
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)

diagnostic_compare_df_628hc3a2 = (
    diagnostic_df_628hc3a2
    .sort_values(
        "scenario_number"
    )
    .reset_index(drop=True)
)

if not np.array_equal(
    uncertainty_only_df_628hc3a2[
        "scenario_id"
    ].to_numpy(),
    diagnostic_compare_df_628hc3a2[
        "scenario_id"
    ].to_numpy()
):
    raise RuntimeError(
        "Scenario alignment failed during gamma comparison."
    )

gamma_vs_uncertainty_628hc3a2 = (
    diagnostic_compare_df_628hc3a2[
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            "full_mae",
            "gain_90",
            "gain_80",
            "gain_70",
        ]
    ]
    .copy()
)

gamma_vs_uncertainty_628hc3a2[
    "uncertainty_gain_90"
] = uncertainty_only_df_628hc3a2[
    "gain_90"
].to_numpy()

gamma_vs_uncertainty_628hc3a2[
    "uncertainty_gain_80"
] = uncertainty_only_df_628hc3a2[
    "gain_80"
].to_numpy()

gamma_vs_uncertainty_628hc3a2[
    "uncertainty_gain_70"
] = uncertainty_only_df_628hc3a2[
    "gain_70"
].to_numpy()

gamma_vs_uncertainty_628hc3a2[
    "delta_gain80_vs_uncertainty"
] = (
    gamma_vs_uncertainty_628hc3a2[
        "gain_80"
    ]
    -
    gamma_vs_uncertainty_628hc3a2[
        "uncertainty_gain_80"
    ]
)

FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2 = (
    gamma_vs_uncertainty_628hc3a2
)

print(
    "\n--- DIAGNOSTIC GAMMA VS UNCERTAINTY-ONLY ---"
)

comparison_summary_628hc3a2 = pd.DataFrame(
    {
        "metric": [
            "median_delta_gain80",
            "mean_delta_gain80",
            "q10_delta_gain80",
            "improved_scenarios",
            "equal_scenarios",
            "worse_scenarios",
        ],
        "value": [
            float(
                FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                    "delta_gain80_vs_uncertainty"
                ].median()
            ),

            float(
                FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                    "delta_gain80_vs_uncertainty"
                ].mean()
            ),

            float(
                FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                    "delta_gain80_vs_uncertainty"
                ].quantile(0.10)
            ),

            int(
                (
                    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                        "delta_gain80_vs_uncertainty"
                    ]
                    >
                    1e-12
                ).sum()
            ),

            int(
                np.isclose(
                    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                        "delta_gain80_vs_uncertainty"
                    ].to_numpy(),
                    0.0,
                    atol=1e-12,
                    rtol=0.0
                ).sum()
            ),

            int(
                (
                    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2[
                        "delta_gain80_vs_uncertainty"
                    ]
                    <
                    -1e-12
                ).sum()
            ),
        ],
    }
)

FD002_C3A2_COMPARISON_SUMMARY_628HC3A2 = (
    comparison_summary_628hc3a2
)

display(
    FD002_C3A2_COMPARISON_SUMMARY_628HC3A2
)


# =====================================================================
# 17. BEST / WORST DELTA CASES
# =====================================================================

print(
    "\n--- BEST 15 SCENARIOS VS UNCERTAINTY-ONLY ---"
)

display(
    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2
    .sort_values(
        "delta_gain80_vs_uncertainty",
        ascending=False
    )
    .head(15)
)

print(
    "\n--- WORST 15 SCENARIOS VS UNCERTAINTY-ONLY ---"
)

display(
    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2
    .sort_values(
        "delta_gain80_vs_uncertainty",
        ascending=True
    )
    .head(15)
)


# =====================================================================
# 18. FINAL AUDIT
# =====================================================================

expected_grid_rows_628hc3a2 = (
    315
    *
    len(
        GAMMA_GRID_628HC3A2
    )
)

audit_rows_628hc3a2 = [
    {
        "check": "scenario_count_315",
        "passed":
            n_scenarios_628hc3a2 == 315,
    },
    {
        "check": "prediction_matrix_shape",
        "passed":
            prediction_matrix_628hc3a2.shape
            ==
            (315, n_dev_628hc3a2),
    },
    {
        "check": "uncertainty_matrix_shape",
        "passed":
            uncertainty_matrix_628hc3a2.shape
            ==
            (315, n_dev_628hc3a2),
    },
    {
        "check": "quality_matrix_shape",
        "passed":
            quality_matrix_628hc3a2.shape
            ==
            (315, n_dev_628hc3a2),
    },
    {
        "check": "gamma_grid_row_count",
        "passed":
            len(
                FD002_C3A2_SCENARIO_METRICS_628HC3A2
            )
            ==
            expected_grid_rows_628hc3a2,
    },
    {
        "check": "scenario_metrics_finite",
        "passed":
            bool(
                np.isfinite(
                    FD002_C3A2_SCENARIO_METRICS_628HC3A2[
                        [
                            "full_mae",
                            "gain_90",
                            "gain_80",
                            "gain_70",
                        ]
                    ].to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
    },
    {
        "check": "lofo_family_count_5",
        "passed":
            len(
                FD002_C3A2_LOFO_SUMMARY_628HC3A2
            )
            ==
            5,
    },
    {
        "check": "selected_gammas_within_0_1",
        "passed":
            bool(
                np.all(
                    (
                        selected_gammas_628hc3a2
                        >=
                        0.0
                    )
                    &
                    (
                        selected_gammas_628hc3a2
                        <=
                        1.0
                    )
                )
            ),
    },
]

FD002_C3A2_FINAL_AUDIT_628HC3A2 = (
    pd.DataFrame(
        audit_rows_628hc3a2
    )
)

print("\n--- FINAL C3-A2 AUDIT ---")
display(
    FD002_C3A2_FINAL_AUDIT_628HC3A2
)

all_checks_passed_628hc3a2 = bool(
    FD002_C3A2_FINAL_AUDIT_628HC3A2[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a2:
    raise RuntimeError(
        "STEP 6.28H-C3-A2 final audit failed."
    )


# =====================================================================
# 19. DIAGNOSTIC CONTRACT
# =====================================================================

FD002_C3A2_DIAGNOSTIC_CONTRACT_628HC3A2 = {
    "purpose":
        "development_only_robust_fusion_stability_evaluation",

    "fusion_formula":
        "U + (1-U) * gamma * normalized_quality_excess",

    "quality_excess_reference":
        "FIT_sensor_quality_q95",

    "quality_fit_q95":
        quality_fit_q95_628hc3a2,

    "gamma_grid":
        GAMMA_GRID_628HC3A2.copy(),

    "coverages":
        COVERAGES_628HC3A2,

    "lofo_families":
        tuple(
            CORRUPTION_FAMILIES_628HC3A2
        ),

    "diagnostic_top_gamma":
        diagnostic_gamma_628hc3a2,

    "fusion_rule_frozen":
        False,

    "combination_weight_frozen":
        False,

    "abstention_threshold_selected":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "model_executed":
        False,

    "model_retrained":
        False,

    "files_written":
        False,
}


# =====================================================================
# 20. FINAL SUMMARY
# =====================================================================

print("\n" + "=" * 86)
print("STEP 6.28H-C3-A2 FINAL SUMMARY")
print("=" * 86)

print(
    "Scenarios evaluated              :",
    n_scenarios_628hc3a2
)
print(
    "DEV windows/scenario             :",
    n_dev_628hc3a2
)
print(
    "Gamma values evaluated           :",
    len(GAMMA_GRID_628HC3A2)
)
print(
    "Scenario-gamma evaluations       :",
    len(
        FD002_C3A2_SCENARIO_METRICS_628HC3A2
    )
)
print(
    "LOFO corruption families         :",
    len(
        FD002_C3A2_LOFO_SUMMARY_628HC3A2
    )
)
print(
    "Diagnostic top gamma             :",
    diagnostic_gamma_628hc3a2
)
print(
    "Median LOFO selected gamma       :",
    float(
        np.median(
            selected_gammas_628hc3a2
        )
    )
)
print(
    "LOFO selected gamma range        :",
    (
        float(
            np.min(
                selected_gammas_628hc3a2
            )
        ),
        float(
            np.max(
                selected_gammas_628hc3a2
            )
        ),
    )
)
print(
    "All C3-A2 checks passed          :",
    all_checks_passed_628hc3a2
)
print(
    "DEV RUL labels used              :",
    True
)
print(
    "Official TEST labels read        :",
    False
)
print(
    "TEST scored                      :",
    False
)
print(
    "Fusion rule frozen               :",
    False
)
print(
    "Combination weight frozen        :",
    False
)
print(
    "Abstention threshold selected    :",
    False
)
print(
    "Model executed                   :",
    False
)
print(
    "Model retrained                  :",
    False
)
print(
    "Files written                    :",
    False
)
print(
    "Runtime (sec)                    :",
    runtime_628hc3a2
)

print("\n=== END STEP 6.28H-C3-A2 ===")

In [ ]:
# =============================================================================
# STEP 6.28H-C3-A3-A0
# ADAPTIVE COMBINED-RELIABILITY FUSION:
# SENSOR-QUALITY BRANCH AVAILABILITY PRE-FLIGHT
#
# PURPOSE
# -------
# Before designing any adaptive fusion rule, audit whether the frozen
# branch-level sensor-quality signals from C2 already exist in memory.
#
# IMPORTANT
# ---------
# - No model execution
# - No model retraining
# - No official TEST labels
# - No DEV RUL/error evaluation
# - No threshold selection
# - No gamma selection
# - No fusion-rule selection
# - No files written
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 88)
print("STEP 6.28H-C3-A3-A0 — ADAPTIVE-FUSION BRANCH AVAILABILITY PRE-FLIGHT")
print("=" * 88)


# =============================================================================
# 1. REQUIRED FROZEN CORE OBJECTS
# =============================================================================

required_core_628hc3a3a0 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "uncertainty_risk_matrix_628hc3a0r1",
    "clean_uncertainty_risk_628hc3a0r1",
]

print("\n--- REQUIRED FROZEN CORE OBJECTS ---")

missing_core_628hc3a3a0 = []

for name in required_core_628hc3a3a0:
    present = name in globals()
    print(f"{name:<52}: {present}")

    if not present:
        missing_core_628hc3a3a0.append(name)

if missing_core_628hc3a3a0:
    raise RuntimeError(
        "Missing required frozen objects: "
        + ", ".join(missing_core_628hc3a3a0)
    )


# =============================================================================
# 2. POPULATION CONTRACT
# =============================================================================

manifest_628hc3a3a0 = FD002_CORRUPTION_MANIFEST_628GD.copy()

n_scenarios_628hc3a3a0 = len(manifest_628hc3a3a0)
n_dev_628hc3a3a0 = len(clean_uncertainty_risk_628hc3a0r1)

quality_matrix_628hc3a3a0 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

uncertainty_matrix_628hc3a3a0 = np.asarray(
    uncertainty_risk_matrix_628hc3a0r1,
    dtype=np.float64
)

clean_uncertainty_628hc3a3a0 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64
).reshape(-1)

print("\n--- POPULATION CONTRACT ---")
print("Scenarios                 :", n_scenarios_628hc3a3a0)
print("DEV windows               :", n_dev_628hc3a3a0)
print("Uncertainty-risk matrix   :", uncertainty_matrix_628hc3a3a0.shape)
print("Sensor-quality risk matrix:", quality_matrix_628hc3a3a0.shape)

expected_shape_628hc3a3a0 = (
    n_scenarios_628hc3a3a0,
    n_dev_628hc3a3a0
)

if uncertainty_matrix_628hc3a3a0.shape != expected_shape_628hc3a3a0:
    raise RuntimeError(
        "Unexpected uncertainty-risk matrix shape."
    )

if quality_matrix_628hc3a3a0.shape != expected_shape_628hc3a3a0:
    raise RuntimeError(
        "Unexpected sensor-quality matrix shape."
    )


# =============================================================================
# 3. CLEAN SENSOR-QUALITY OBJECT INSPECTION
# =============================================================================

clean_quality_obj_628hc3a3a0 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

print("\n--- CLEAN SENSOR-QUALITY OBJECT ---")
print("Type:", type(clean_quality_obj_628hc3a3a0).__name__)

if not isinstance(clean_quality_obj_628hc3a3a0, dict):
    raise RuntimeError(
        "Expected CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B "
        "to be a dictionary."
    )

print("\nDictionary keys:")

clean_key_rows_628hc3a3a0 = []

for key, value in clean_quality_obj_628hc3a3a0.items():

    try:
        arr = np.asarray(value)

        shape = arr.shape
        length = len(arr) if arr.ndim > 0 else None
        finite = (
            bool(np.isfinite(arr.astype(np.float64)).all())
            if np.issubdtype(arr.dtype, np.number)
            else None
        )

    except Exception:
        shape = None
        length = None
        finite = None

    clean_key_rows_628hc3a3a0.append({
        "key": key,
        "type": type(value).__name__,
        "shape": shape,
        "length": length,
        "finite": finite,
    })

display(pd.DataFrame(clean_key_rows_628hc3a3a0))


# =============================================================================
# 4. VERIFY THE FOUR CLEAN BRANCHES
# =============================================================================

expected_branches_628hc3a3a0 = [
    "level",
    "noise",
    "dropout",
    "stuck",
]

print("\n--- CLEAN FOUR-BRANCH CONTRACT ---")

clean_branch_vectors_628hc3a3a0 = {}

for branch in expected_branches_628hc3a3a0:

    exists = branch in clean_quality_obj_628hc3a3a0

    if not exists:
        print(f"{branch:<10}: MISSING")
        continue

    arr = np.asarray(
        clean_quality_obj_628hc3a3a0[branch],
        dtype=np.float64
    ).reshape(-1)

    clean_branch_vectors_628hc3a3a0[branch] = arr

    print(
        f"{branch:<10}: "
        f"shape={arr.shape} | "
        f"finite={np.isfinite(arr).all()} | "
        f"within_[0,1]={np.all((arr >= 0.0) & (arr <= 1.0))}"
    )

clean_four_branch_exact_628hc3a3a0 = (
    len(clean_branch_vectors_628hc3a3a0) == 4
    and all(
        len(v) == n_dev_628hc3a3a0
        for v in clean_branch_vectors_628hc3a3a0.values()
    )
)

print(
    "\nComplete clean four-branch contract:",
    clean_four_branch_exact_628hc3a3a0
)


# =============================================================================
# 5. SEARCH CURRENT NAMESPACE FOR CORRUPTED BRANCH MATRICES
#
# We deliberately DO NOT create or recompute anything here.
# We first look for reusable frozen objects.
# =============================================================================

branch_keywords_628hc3a3a0 = [
    "level",
    "noise",
    "dropout",
    "stuck",
]

candidate_rows_628hc3a3a0 = []


# -------------------------------------------------------------------------
# A. Direct ndarray / DataFrame matrices
# -------------------------------------------------------------------------

for name, obj in list(globals().items()):

    lname = str(name).lower()

    if not any(k in lname for k in branch_keywords_628hc3a3a0):
        continue

    if not any(
        marker in lname
        for marker in [
            "quality",
            "risk",
            "branch",
            "sensor",
            "628hc2",
            "628hc3",
        ]
    ):
        continue

    try:
        if isinstance(obj, pd.DataFrame):
            arr = obj.to_numpy()

        elif isinstance(obj, np.ndarray):
            arr = obj

        else:
            continue

        if arr.shape == expected_shape_628hc3a3a0:

            candidate_rows_628hc3a3a0.append({
                "object_name": name,
                "container": type(obj).__name__,
                "candidate_branch": next(
                    (
                        b for b in branch_keywords_628hc3a3a0
                        if b in lname
                    ),
                    "unknown"
                ),
                "shape": arr.shape,
                "finite": (
                    bool(np.isfinite(
                        np.asarray(arr, dtype=np.float64)
                    ).all())
                ),
                "within_0_1": (
                    bool(np.all(
                        (np.asarray(arr, dtype=np.float64) >= 0.0)
                        &
                        (np.asarray(arr, dtype=np.float64) <= 1.0)
                    ))
                ),
            })

    except Exception:
        pass


# =============================================================================
# 6. SEARCH DICTIONARY OBJECTS FOR 315 x 8367 BRANCH MATRICES
# =============================================================================

dict_candidate_rows_628hc3a3a0 = []

for name, obj in list(globals().items()):

    if not isinstance(obj, dict):
        continue

    matching_keys = []

    for branch in branch_keywords_628hc3a3a0:

        if branch not in obj:
            continue

        try:
            arr = np.asarray(obj[branch])

            if arr.shape == expected_shape_628hc3a3a0:
                matching_keys.append(branch)

        except Exception:
            pass

    if matching_keys:

        dict_candidate_rows_628hc3a3a0.append({
            "object_name": name,
            "matching_branch_keys": matching_keys,
            "matching_count": len(matching_keys),
        })


# =============================================================================
# 7. SEARCH LIST/TUPLE OF 315 PER-SCENARIO DICTIONARIES
# =============================================================================

sequence_candidate_rows_628hc3a3a0 = []

for name, obj in list(globals().items()):

    if not isinstance(obj, (list, tuple)):
        continue

    if len(obj) != n_scenarios_628hc3a3a0:
        continue

    if not all(isinstance(x, dict) for x in obj[:min(5, len(obj))]):
        continue

    common_branch_keys = set(expected_branches_628hc3a3a0)

    for x in obj[:min(20, len(obj))]:
        common_branch_keys &= set(x.keys())

    if common_branch_keys:

        sequence_candidate_rows_628hc3a3a0.append({
            "object_name": name,
            "type": type(obj).__name__,
            "length": len(obj),
            "common_branch_keys": sorted(common_branch_keys),
        })


# =============================================================================
# 8. DISPLAY DISCOVERED CANDIDATES
# =============================================================================

print("\n--- DIRECT CORRUPTED BRANCH-MATRIX CANDIDATES ---")

direct_candidates_df_628hc3a3a0 = pd.DataFrame(
    candidate_rows_628hc3a3a0
)

if len(direct_candidates_df_628hc3a3a0):
    display(
        direct_candidates_df_628hc3a3a0
        .sort_values(
            ["candidate_branch", "object_name"]
        )
        .reset_index(drop=True)
    )
else:
    print("None found.")


print("\n--- DICTIONARY CORRUPTED-BRANCH CANDIDATES ---")

dict_candidates_df_628hc3a3a0 = pd.DataFrame(
    dict_candidate_rows_628hc3a3a0
)

if len(dict_candidates_df_628hc3a3a0):
    display(dict_candidates_df_628hc3a3a0)
else:
    print("None found.")


print("\n--- PER-SCENARIO SEQUENCE CANDIDATES ---")

sequence_candidates_df_628hc3a3a0 = pd.DataFrame(
    sequence_candidate_rows_628hc3a3a0
)

if len(sequence_candidates_df_628hc3a3a0):
    display(sequence_candidates_df_628hc3a3a0)
else:
    print("None found.")


# =============================================================================
# 9. INSPECT IMPORTANT C2 RESPONSE OBJECTS WITHOUT MODIFYING THEM
# =============================================================================

inspect_names_628hc3a3a0 = [
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
    "FIT_SENSOR_QUALITY_628HC2A",
    "CLEAN_DEV_SENSOR_QUALITY_628HC2A",
    "FD002_SENSOR_QUALITY_REFERENCE_628HC2A",
    "sensor_quality_branches_628hc2a",
]

inspection_rows_628hc3a3a0 = []

for name in inspect_names_628hc3a3a0:

    if name not in globals():
        inspection_rows_628hc3a3a0.append({
            "object_name": name,
            "exists": False,
            "type": None,
            "shape_or_length": None,
            "keys_or_columns": None,
        })
        continue

    obj = globals()[name]

    if isinstance(obj, pd.DataFrame):
        shape_or_length = obj.shape
        keys_or_columns = list(obj.columns)

    elif isinstance(obj, dict):
        shape_or_length = len(obj)
        keys_or_columns = list(obj.keys())

    elif isinstance(obj, np.ndarray):
        shape_or_length = obj.shape
        keys_or_columns = None

    elif isinstance(obj, (list, tuple)):
        shape_or_length = len(obj)
        keys_or_columns = None

    else:
        shape_or_length = None
        keys_or_columns = None

    inspection_rows_628hc3a3a0.append({
        "object_name": name,
        "exists": True,
        "type": type(obj).__name__,
        "shape_or_length": shape_or_length,
        "keys_or_columns": keys_or_columns,
    })


print("\n--- RELEVANT C2 OBJECT INSPECTION ---")
display(pd.DataFrame(inspection_rows_628hc3a3a0))


# =============================================================================
# 10. SAFETY / EXPERIMENT BOUNDARY AUDIT
# =============================================================================

audit_628hc3a3a0 = pd.DataFrame([
    {
        "check": "scenario_count_315",
        "passed": n_scenarios_628hc3a3a0 == 315,
    },
    {
        "check": "uncertainty_matrix_shape",
        "passed":
            uncertainty_matrix_628hc3a3a0.shape
            == expected_shape_628hc3a3a0,
    },
    {
        "check": "quality_matrix_shape",
        "passed":
            quality_matrix_628hc3a3a0.shape
            == expected_shape_628hc3a3a0,
    },
    {
        "check": "clean_four_branch_contract",
        "passed": clean_four_branch_exact_628hc3a3a0,
    },
    {
        "check": "uncertainty_matrix_finite",
        "passed":
            bool(np.isfinite(
                uncertainty_matrix_628hc3a3a0
            ).all()),
    },
    {
        "check": "quality_matrix_finite",
        "passed":
            bool(np.isfinite(
                quality_matrix_628hc3a3a0
            ).all()),
    },
])

print("\n--- C3-A3-A0 AUDIT ---")
display(audit_628hc3a3a0)


# =============================================================================
# 11. FINAL SUMMARY
# =============================================================================

all_preflight_checks_628hc3a3a0 = bool(
    audit_628hc3a3a0["passed"].all()
)

direct_branch_matrix_count_628hc3a3a0 = (
    len(direct_candidates_df_628hc3a3a0)
)

dict_branch_container_count_628hc3a3a0 = (
    len(dict_candidates_df_628hc3a3a0)
)

sequence_branch_container_count_628hc3a3a0 = (
    len(sequence_candidates_df_628hc3a3a0)
)

print("\n" + "=" * 88)
print("STEP 6.28H-C3-A3-A0 FINAL SUMMARY")
print("=" * 88)

print(
    f"Scenarios                              : "
    f"{n_scenarios_628hc3a3a0}"
)
print(
    f"DEV windows                            : "
    f"{n_dev_628hc3a3a0}"
)
print(
    f"Clean four quality branches available  : "
    f"{clean_four_branch_exact_628hc3a3a0}"
)
print(
    f"Direct corrupted branch matrices found : "
    f"{direct_branch_matrix_count_628hc3a3a0}"
)
print(
    f"Dictionary branch containers found     : "
    f"{dict_branch_container_count_628hc3a3a0}"
)
print(
    f"Sequence branch containers found       : "
    f"{sequence_branch_container_count_628hc3a3a0}"
)
print(
    f"All pre-flight checks passed           : "
    f"{all_preflight_checks_628hc3a3a0}"
)

print("DEV RUL labels used                    : False")
print("Official TEST labels read              : False")
print("TEST scored                            : False")
print("Adaptive rule selected                 : False")
print("Adaptive threshold selected            : False")
print("Combination weight selected            : False")
print("Model executed                         : False")
print("Model retrained                        : False")
print("Files written                          : False")

print("\n=== END STEP 6.28H-C3-A3-A0 ===")

In [ ]:
# =============================================================================
# STEP 6.28H-C3-A3-A1
# FROZEN C2 SENSOR-QUALITY BRANCH FUNCTION CONTRACT INSPECTION
#
# PURPOSE
# -------
# Inspect the exact frozen C2 branch function, calibration/reference objects,
# and response-table structure BEFORE reconstructing corrupted branch matrices.
#
# IMPORTANT
# ---------
# - No corruption rerun
# - No model execution
# - No model retraining
# - No DEV RUL/error evaluation
# - No TEST labels
# - No fusion-rule selection
# - No thresholds selected
# - No files written
# =============================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 92)
print("STEP 6.28H-C3-A3-A1 — FROZEN C2 BRANCH-FUNCTION CONTRACT INSPECTION")
print("=" * 92)


# =============================================================================
# 1. REQUIRED OBJECTS
# =============================================================================

required_628hc3a3a1 = [
    "sensor_quality_branches_628hc2a",
    "FIT_SENSOR_QUALITY_628HC2A",
    "CLEAN_DEV_SENSOR_QUALITY_628HC2A",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FD002_SENSOR_QUALITY_REFERENCE_628HC2A",
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc3a3a1 = []

for name in required_628hc3a3a1:
    present = name in globals()
    print(f"{name:<52}: {present}")

    if not present:
        missing_628hc3a3a1.append(name)

if missing_628hc3a3a1:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc3a3a1)
    )


# =============================================================================
# 2. FUNCTION CONTRACT
# =============================================================================

branch_fn_628hc3a3a1 = sensor_quality_branches_628hc2a

print("\n--- FROZEN BRANCH FUNCTION CONTRACT ---")
print("Function name :", branch_fn_628hc3a3a1.__name__)

try:
    print("Signature     :", inspect.signature(branch_fn_628hc3a3a1))
except Exception as exc:
    print("Signature     : unavailable")
    print("Reason        :", repr(exc))


# =============================================================================
# 3. FUNCTION SOURCE
#
# Printing the source is intentional here:
# we need to know exactly which reference objects and calculations the
# frozen C2 implementation uses before reproducing it over corruptions.
# =============================================================================

print("\n--- FROZEN BRANCH FUNCTION SOURCE ---")

try:
    source_628hc3a3a1 = inspect.getsource(branch_fn_628hc3a3a1)
    print(source_628hc3a3a1)

except Exception as exc:
    source_628hc3a3a1 = None
    print("Source unavailable.")
    print("Reason:", repr(exc))


# =============================================================================
# 4. FIT SENSOR-QUALITY OBJECT
# =============================================================================

fit_quality_628hc3a3a1 = FIT_SENSOR_QUALITY_628HC2A

print("\n--- FIT_SENSOR_QUALITY_628HC2A ---")
print("Type:", type(fit_quality_628hc3a3a1).__name__)

if isinstance(fit_quality_628hc3a3a1, dict):

    fit_rows_628hc3a3a1 = []

    for key, value in fit_quality_628hc3a3a1.items():

        try:
            arr = np.asarray(value)

            fit_rows_628hc3a3a1.append({
                "key": key,
                "type": type(value).__name__,
                "shape": arr.shape,
                "dtype": str(arr.dtype),
                "minimum": (
                    float(np.nanmin(arr))
                    if np.issubdtype(arr.dtype, np.number)
                    else None
                ),
                "median": (
                    float(np.nanmedian(arr))
                    if np.issubdtype(arr.dtype, np.number)
                    else None
                ),
                "maximum": (
                    float(np.nanmax(arr))
                    if np.issubdtype(arr.dtype, np.number)
                    else None
                ),
            })

        except Exception:
            fit_rows_628hc3a3a1.append({
                "key": key,
                "type": type(value).__name__,
                "shape": None,
                "dtype": None,
                "minimum": None,
                "median": None,
                "maximum": None,
            })

    display(pd.DataFrame(fit_rows_628hc3a3a1))

else:
    print("Unexpected type.")


# =============================================================================
# 5. CLEAN RAW SENSOR-QUALITY OBJECT
# =============================================================================

clean_raw_quality_628hc3a3a1 = CLEAN_DEV_SENSOR_QUALITY_628HC2A

print("\n--- CLEAN_DEV_SENSOR_QUALITY_628HC2A ---")
print("Type:", type(clean_raw_quality_628hc3a3a1).__name__)

if isinstance(clean_raw_quality_628hc3a3a1, dict):

    clean_raw_rows_628hc3a3a1 = []

    for key, value in clean_raw_quality_628hc3a3a1.items():

        try:
            arr = np.asarray(value, dtype=np.float64).reshape(-1)

            clean_raw_rows_628hc3a3a1.append({
                "key": key,
                "shape": arr.shape,
                "finite": bool(np.isfinite(arr).all()),
                "minimum": float(np.min(arr)),
                "median": float(np.median(arr)),
                "mean": float(np.mean(arr)),
                "maximum": float(np.max(arr)),
            })

        except Exception:
            clean_raw_rows_628hc3a3a1.append({
                "key": key,
                "shape": None,
                "finite": None,
                "minimum": None,
                "median": None,
                "mean": None,
                "maximum": None,
            })

    display(pd.DataFrame(clean_raw_rows_628hc3a3a1))


# =============================================================================
# 6. CALIBRATED CLEAN SENSOR-QUALITY OBJECT
# =============================================================================

clean_cal_quality_628hc3a3a1 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

print("\n--- CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B ---")
print("Type:", type(clean_cal_quality_628hc3a3a1).__name__)

cal_rows_628hc3a3a1 = []

for key, value in clean_cal_quality_628hc3a3a1.items():

    try:
        arr = np.asarray(value, dtype=np.float64).reshape(-1)

        cal_rows_628hc3a3a1.append({
            "key": key,
            "shape": arr.shape,
            "finite": bool(np.isfinite(arr).all()),
            "minimum": float(np.min(arr)),
            "median": float(np.median(arr)),
            "mean": float(np.mean(arr)),
            "maximum": float(np.max(arr)),
        })

    except Exception:
        cal_rows_628hc3a3a1.append({
            "key": key,
            "shape": None,
            "finite": None,
            "minimum": None,
            "median": None,
            "mean": None,
            "maximum": None,
        })

display(pd.DataFrame(cal_rows_628hc3a3a1))


# =============================================================================
# 7. REFERENCE OBJECT STRUCTURE
# =============================================================================

reference_628hc3a3a1 = FD002_SENSOR_QUALITY_REFERENCE_628HC2A

print("\n--- FD002_SENSOR_QUALITY_REFERENCE_628HC2A ---")
print("Type:", type(reference_628hc3a3a1).__name__)

reference_rows_628hc3a3a1 = []

if isinstance(reference_628hc3a3a1, dict):

    for key, value in reference_628hc3a3a1.items():

        row = {
            "key": key,
            "type": type(value).__name__,
            "shape_or_length": None,
            "preview": None,
        }

        try:

            if isinstance(value, np.ndarray):
                row["shape_or_length"] = value.shape

                if value.size <= 10:
                    row["preview"] = value.tolist()
                else:
                    row["preview"] = (
                        f"min={np.nanmin(value):.6g}, "
                        f"median={np.nanmedian(value):.6g}, "
                        f"max={np.nanmax(value):.6g}"
                    )

            elif isinstance(value, pd.DataFrame):
                row["shape_or_length"] = value.shape
                row["preview"] = list(value.columns)

            elif isinstance(value, pd.Series):
                row["shape_or_length"] = value.shape
                row["preview"] = value.head().tolist()

            elif isinstance(value, dict):
                row["shape_or_length"] = len(value)
                row["preview"] = list(value.keys())

            elif isinstance(value, (list, tuple)):
                row["shape_or_length"] = len(value)
                row["preview"] = (
                    value
                    if len(value) <= 10
                    else list(value[:5])
                )

            else:
                row["preview"] = repr(value)

        except Exception as exc:
            row["preview"] = f"<inspection failed: {exc!r}>"

        reference_rows_628hc3a3a1.append(row)

else:
    reference_rows_628hc3a3a1.append({
        "key": "<object>",
        "type": type(reference_628hc3a3a1).__name__,
        "shape_or_length": None,
        "preview": repr(reference_628hc3a3a1),
    })

reference_table_628hc3a3a1 = pd.DataFrame(
    reference_rows_628hc3a3a1
)

display(reference_table_628hc3a3a1)


# =============================================================================
# 8. FULL C2-C RESPONSE TABLE CONTRACT
# =============================================================================

response_628hc3a3a1 = FD002_SENSOR_QUALITY_RESPONSE_628HC2C

print("\n--- FD002_SENSOR_QUALITY_RESPONSE_628HC2C ---")
print("Shape:", response_628hc3a3a1.shape)

print("\nColumns:")
for i, col in enumerate(response_628hc3a3a1.columns):
    print(f"{i:02d}: {col}")

print("\nFirst 3 rows:")
display(response_628hc3a3a1.head(3))


# =============================================================================
# 9. SEARCH FOR OTHER C2 SENSOR-QUALITY FUNCTIONS
# =============================================================================

print("\n--- RELEVANT SENSOR-QUALITY CALLABLES ---")

callable_rows_628hc3a3a1 = []

for name, obj in list(globals().items()):

    lname = str(name).lower()

    if not callable(obj):
        continue

    if not (
        "quality" in lname
        or "628hc2" in lname
        or "calibr" in lname
    ):
        continue

    try:
        signature = str(inspect.signature(obj))
    except Exception:
        signature = "<unavailable>"

    callable_rows_628hc3a3a1.append({
        "function_name": name,
        "signature": signature,
    })

callables_df_628hc3a3a1 = pd.DataFrame(
    callable_rows_628hc3a3a1
)

if len(callables_df_628hc3a3a1):
    display(
        callables_df_628hc3a3a1
        .sort_values("function_name")
        .reset_index(drop=True)
    )
else:
    print("None found.")


# =============================================================================
# 10. AUDIT
# =============================================================================

expected_clean_keys_628hc3a3a1 = {
    "level",
    "noise",
    "dropout",
    "stuck",
}

audit_628hc3a3a1 = pd.DataFrame([
    {
        "check": "branch_function_callable",
        "passed": callable(branch_fn_628hc3a3a1),
    },
    {
        "check": "fit_quality_dictionary",
        "passed": isinstance(fit_quality_628hc3a3a1, dict),
    },
    {
        "check": "clean_raw_dictionary",
        "passed": isinstance(clean_raw_quality_628hc3a3a1, dict),
    },
    {
        "check": "clean_calibrated_dictionary",
        "passed": isinstance(clean_cal_quality_628hc3a3a1, dict),
    },
    {
        "check": "reference_dictionary",
        "passed": isinstance(reference_628hc3a3a1, dict),
    },
    {
        "check": "clean_raw_four_branches",
        "passed":
            expected_clean_keys_628hc3a3a1
            .issubset(clean_raw_quality_628hc3a3a1.keys()),
    },
    {
        "check": "clean_calibrated_four_branches",
        "passed":
            expected_clean_keys_628hc3a3a1
            .issubset(clean_cal_quality_628hc3a3a1.keys()),
    },
    {
        "check": "c2_response_315_rows",
        "passed": len(response_628hc3a3a1) == 315,
    },
])

print("\n--- C3-A3-A1 AUDIT ---")
display(audit_628hc3a3a1)


# =============================================================================
# 11. FINAL SUMMARY
# =============================================================================

all_checks_628hc3a3a1 = bool(
    audit_628hc3a3a1["passed"].all()
)

print("\n" + "=" * 92)
print("STEP 6.28H-C3-A3-A1 FINAL SUMMARY")
print("=" * 92)

print(
    "Frozen branch function callable       :",
    callable(branch_fn_628hc3a3a1),
)
print(
    "Branch function signature inspected   : True"
)
print(
    "FIT quality reference inspected       :",
    isinstance(fit_quality_628hc3a3a1, dict),
)
print(
    "C2 reference contract inspected       :",
    isinstance(reference_628hc3a3a1, dict),
)
print(
    "Clean raw four branches available     :",
    expected_clean_keys_628hc3a3a1.issubset(
        clean_raw_quality_628hc3a3a1.keys()
    ),
)
print(
    "Clean calibrated branches available   :",
    expected_clean_keys_628hc3a3a1.issubset(
        clean_cal_quality_628hc3a3a1.keys()
    ),
)
print(
    "C2-C scenario response rows           :",
    len(response_628hc3a3a1),
)
print(
    "All A1 contract checks passed         :",
    all_checks_628hc3a3a1,
)

print("Corrupted branch matrices rebuilt     : False")
print("DEV RUL labels used                   : False")
print("Official TEST labels read             : False")
print("TEST scored                           : False")
print("Adaptive fusion rule selected         : False")
print("Adaptive threshold selected           : False")
print("Model executed                        : False")
print("Model retrained                       : False")
print("Files written                         : False")

print("\n=== END STEP 6.28H-C3-A3-A1 ===")

In [ ]:
# =============================================================================
# STEP 6.28H-C3-A3-A2-A0
# CORRUPTED FOUR-BRANCH RECONSTRUCTION CONTRACT PRE-FLIGHT
#
# PURPOSE
# -------
# Verify the exact frozen calibration and corruption pipeline that will be used
# to reconstruct the four calibrated sensor-quality branch matrices.
#
# NO:
# - DEV RUL labels
# - TEST labels
# - model execution
# - model retraining
# - fusion-rule selection
# - threshold selection
# - file writing
# =============================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 96)
print("STEP 6.28H-C3-A3-A2-A0 — FOUR-BRANCH RECONSTRUCTION CONTRACT PRE-FLIGHT")
print("=" * 96)


# =============================================================================
# 1. REQUIRED OBJECTS
# =============================================================================

required_628hc3a3a2a0 = [
    "sensor_quality_branches_628hc2a",
    "calibrate_branch_scores_628hc2b",
    "final_sensor_quality_from_raw_628hc2b",
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",
    "X_DEV_FD002_628D",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc3a3a2a0 = []

for name in required_628hc3a3a2a0:
    present = name in globals()
    print(f"{name:<52}: {present}")

    if not present:
        missing_628hc3a3a2a0.append(name)

if missing_628hc3a3a2a0:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc3a3a2a0)
    )


# =============================================================================
# 2. EXACT FUNCTION SIGNATURES
# =============================================================================

functions_628hc3a3a2a0 = {
    "sensor_quality_branches_628hc2a":
        sensor_quality_branches_628hc2a,

    "calibrate_branch_scores_628hc2b":
        calibrate_branch_scores_628hc2b,

    "final_sensor_quality_from_raw_628hc2b":
        final_sensor_quality_from_raw_628hc2b,

    "apply_scenario_628hc0":
        apply_scenario_628hc0,

    "normalize_corrupted_sensor_628ha":
        normalize_corrupted_sensor_628ha,

    "build_windows_628hc0":
        build_windows_628hc0,
}

print("\n--- EXACT FUNCTION SIGNATURES ---")

signature_rows_628hc3a3a2a0 = []

for name, fn in functions_628hc3a3a2a0.items():

    try:
        sig = str(inspect.signature(fn))
    except Exception:
        sig = "<unavailable>"

    signature_rows_628hc3a3a2a0.append({
        "function": name,
        "signature": sig,
    })

display(pd.DataFrame(signature_rows_628hc3a3a2a0))


# =============================================================================
# 3. CALIBRATION FUNCTION SOURCE
# =============================================================================

print("\n--- calibrate_branch_scores_628hc2b SOURCE ---")

try:
    print(
        inspect.getsource(
            calibrate_branch_scores_628hc2b
        )
    )
except Exception as exc:
    print("Source unavailable:", repr(exc))


print("\n--- final_sensor_quality_from_raw_628hc2b SOURCE ---")

try:
    print(
        inspect.getsource(
            final_sensor_quality_from_raw_628hc2b
        )
    )
except Exception as exc:
    print("Source unavailable:", repr(exc))


# =============================================================================
# 4. CORRUPTION PIPELINE FUNCTION SOURCES
# =============================================================================

print("\n--- apply_scenario_628hc0 SOURCE ---")

try:
    print(
        inspect.getsource(
            apply_scenario_628hc0
        )
    )
except Exception as exc:
    print("Source unavailable:", repr(exc))


print("\n--- normalize_corrupted_sensor_628ha SOURCE ---")

try:
    print(
        inspect.getsource(
            normalize_corrupted_sensor_628ha
        )
    )
except Exception as exc:
    print("Source unavailable:", repr(exc))


print("\n--- build_windows_628hc0 SOURCE ---")

try:
    print(
        inspect.getsource(
            build_windows_628hc0
        )
    )
except Exception as exc:
    print("Source unavailable:", repr(exc))


# =============================================================================
# 5. CLEAN BRANCH REPRODUCTION
#
# Re-run ONLY the label-free branch computation on the already existing
# clean DEV tensor. No model or RUL labels are involved.
# =============================================================================

print("\n--- CLEAN RAW BRANCH REPRODUCTION ---")

clean_raw_rebuilt_628hc3a3a2a0 = (
    sensor_quality_branches_628hc2a(
        X_DEV_FD002_628D
    )
)

print(
    "Return type:",
    type(clean_raw_rebuilt_628hc3a3a2a0).__name__,
)

print(
    "Keys:",
    list(clean_raw_rebuilt_628hc3a3a2a0.keys()),
)


# =============================================================================
# 6. CLEAN CALIBRATION REPRODUCTION
# =============================================================================

print("\n--- CLEAN CALIBRATED BRANCH REPRODUCTION ---")

clean_cal_rebuilt_628hc3a3a2a0 = (
    calibrate_branch_scores_628hc2b(
        clean_raw_rebuilt_628hc3a3a2a0
    )
)

print(
    "Return type:",
    type(clean_cal_rebuilt_628hc3a3a2a0).__name__,
)

if isinstance(clean_cal_rebuilt_628hc3a3a2a0, dict):
    print(
        "Keys:",
        list(clean_cal_rebuilt_628hc3a3a2a0.keys()),
    )


# =============================================================================
# 7. EXACT CLEAN BRANCH COMPARISON
# =============================================================================

branch_names_628hc3a3a2a0 = [
    "level",
    "noise",
    "dropout",
    "stuck",
]

comparison_rows_628hc3a3a2a0 = []

for branch in branch_names_628hc3a3a2a0:

    stored = np.asarray(
        CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
            branch
        ],
        dtype=np.float64,
    ).reshape(-1)

    rebuilt = np.asarray(
        clean_cal_rebuilt_628hc3a3a2a0[
            branch
        ],
        dtype=np.float64,
    ).reshape(-1)

    max_error = float(
        np.max(
            np.abs(
                stored - rebuilt
            )
        )
    )

    comparison_rows_628hc3a3a2a0.append({
        "branch": branch,
        "stored_shape": stored.shape,
        "rebuilt_shape": rebuilt.shape,
        "max_abs_error": max_error,
        "exact_le_1e12": max_error <= 1e-12,
    })

comparison_df_628hc3a3a2a0 = pd.DataFrame(
    comparison_rows_628hc3a3a2a0
)

print("\n--- CLEAN BRANCH EXACTNESS AUDIT ---")
display(comparison_df_628hc3a3a2a0)


# =============================================================================
# 8. FINAL SENSOR-QUALITY FUNCTION RETURN CONTRACT
# =============================================================================

print("\n--- FINAL SENSOR-QUALITY RETURN CONTRACT ---")

clean_final_rebuilt_628hc3a3a2a0 = (
    final_sensor_quality_from_raw_628hc2b(
        clean_raw_rebuilt_628hc3a3a2a0
    )
)

print(
    "Return type:",
    type(clean_final_rebuilt_628hc3a3a2a0).__name__,
)

if isinstance(clean_final_rebuilt_628hc3a3a2a0, dict):

    print(
        "Keys:",
        list(clean_final_rebuilt_628hc3a3a2a0.keys()),
    )

    for key, value in clean_final_rebuilt_628hc3a3a2a0.items():

        try:
            arr = np.asarray(
                value,
                dtype=np.float64,
            )

            print(
                f"{key:<24} "
                f"shape={arr.shape} "
                f"finite={np.isfinite(arr).all()} "
                f"min={np.min(arr):.6f} "
                f"median={np.median(arr):.6f} "
                f"max={np.max(arr):.6f}"
            )

        except Exception:
            print(
                f"{key:<24} "
                f"type={type(value).__name__}"
            )

else:

    try:
        arr = np.asarray(
            clean_final_rebuilt_628hc3a3a2a0,
            dtype=np.float64,
        )

        print(
            "Shape:",
            arr.shape,
        )

        print(
            "Finite:",
            bool(np.isfinite(arr).all()),
        )

    except Exception as exc:
        print(
            "Could not convert return object:",
            repr(exc),
        )


# =============================================================================
# 9. DATA CONTRACTS
# =============================================================================

manifest_628hc3a3a2a0 = (
    FD002_CORRUPTION_MANIFEST_628GD
)

print("\n--- CORRUPTION MANIFEST CONTRACT ---")
print("Type :", type(manifest_628hc3a3a2a0).__name__)
print("Shape:", manifest_628hc3a3a2a0.shape)

print("\nColumns:")
print(list(manifest_628hc3a3a2a0.columns))

print("\nFirst 3 scenarios:")
display(
    manifest_628hc3a3a2a0.head(3)
)


print("\n--- CORRUPTION BASE CONTRACT ---")
print(
    "Type:",
    type(FD002_DEV_CORRUPTION_BASE_628GC).__name__,
)

try:
    print(
        "Shape:",
        FD002_DEV_CORRUPTION_BASE_628GC.shape,
    )
except Exception:
    print(
        "Length:",
        len(FD002_DEV_CORRUPTION_BASE_628GC),
    )


# =============================================================================
# 10. PRE-FLIGHT AUDIT
# =============================================================================

branch_exact_628hc3a3a2a0 = bool(
    comparison_df_628hc3a3a2a0[
        "exact_le_1e12"
    ].all()
)

audit_628hc3a3a2a0 = pd.DataFrame([
    {
        "check": "manifest_315_rows",
        "passed":
            len(manifest_628hc3a3a2a0) == 315,
    },
    {
        "check": "clean_dev_8367_windows",
        "passed":
            X_DEV_FD002_628D.shape[0] == 8367,
    },
    {
        "check": "raw_branch_dictionary",
        "passed":
            isinstance(
                clean_raw_rebuilt_628hc3a3a2a0,
                dict,
            ),
    },
    {
        "check": "calibrated_branch_dictionary",
        "passed":
            isinstance(
                clean_cal_rebuilt_628hc3a3a2a0,
                dict,
            ),
    },
    {
        "check": "four_clean_branches_exact",
        "passed":
            branch_exact_628hc3a3a2a0,
    },
])

print("\n--- C3-A3-A2-A0 AUDIT ---")
display(audit_628hc3a3a2a0)


# =============================================================================
# 11. FINAL SUMMARY
# =============================================================================

all_checks_628hc3a3a2a0 = bool(
    audit_628hc3a3a2a0[
        "passed"
    ].all()
)

print("\n" + "=" * 96)
print("STEP 6.28H-C3-A3-A2-A0 FINAL SUMMARY")
print("=" * 96)

print(
    "Manifest scenarios                  :",
    len(manifest_628hc3a3a2a0),
)

print(
    "Clean DEV windows                  :",
    X_DEV_FD002_628D.shape[0],
)

print(
    "Frozen raw branch function         : True"
)

print(
    "Frozen calibration function        : True"
)

print(
    "Clean four calibrated branches exact:",
    branch_exact_628hc3a3a2a0,
)

print(
    "All reconstruction pre-flight checks:",
    all_checks_628hc3a3a2a0,
)

print("Corrupted branch matrices rebuilt  : False")
print("DEV RUL labels used                : False")
print("Official TEST labels read          : False")
print("TEST scored                        : False")
print("Adaptive fusion rule selected      : False")
print("Adaptive threshold selected        : False")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files written                      : False")

print("\n=== END STEP 6.28H-C3-A3-A2-A0 ===")

In [ ]:
# =============================================================================
# STEP 6.28H-C3-A3-A2
# FULL FOUR-BRANCH CORRUPTED SENSOR-QUALITY MATRIX RECONSTRUCTION
#
# PURPOSE
# -------
# Reconstruct the four FIT-calibrated sensor-quality branch risks for every
# corrupted DEV scenario and every DEV window:
#
#   level
#   noise
#   dropout
#   stuck
#
# The reconstruction uses exactly the frozen C2 pipeline:
#
#   raw corruption
#       -> frozen normalization
#       -> frozen 30-cycle DEV windows
#       -> frozen raw quality branches
#       -> frozen FIT calibration
#
# As a correctness audit, the reconstructed final scalar sensor-quality risk
# must reproduce the already-frozen C2-C risk matrix row-by-row.
#
# IMPORTANT
# ---------
# NO:
# - DEV RUL labels
# - official TEST labels
# - TEST scoring
# - model execution
# - model retraining
# - adaptive-rule selection
# - threshold selection
# - file writing
# =============================================================================

import time
import numpy as np
import pandas as pd


print("=" * 100)
print(
    "STEP 6.28H-C3-A3-A2 — FULL FOUR-BRANCH "
    "CORRUPTED SENSOR-QUALITY RECONSTRUCTION"
)
print("=" * 100)


# =============================================================================
# 1. REQUIRED OBJECTS
# =============================================================================

required_628hc3a3a2 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "X_DEV_FD002_628D",

    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "build_windows_628hc0",

    "sensor_quality_branches_628hc2a",
    "calibrate_branch_scores_628hc2b",
    "final_sensor_quality_from_raw_628hc2b",

    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
]

print("\n--- REQUIRED OBJECTS ---")

missing_628hc3a3a2 = []

for name in required_628hc3a3a2:

    present = name in globals()

    print(
        f"{name:<55}: {present}"
    )

    if not present:
        missing_628hc3a3a2.append(name)

if missing_628hc3a3a2:

    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc3a3a2)
    )


# =============================================================================
# 2. FROZEN POPULATION CONTRACT
# =============================================================================

manifest_628hc3a3a2 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .reset_index(drop=True)
)

n_scenarios_628hc3a3a2 = len(
    manifest_628hc3a3a2
)

n_dev_628hc3a3a2 = (
    X_DEV_FD002_628D.shape[0]
)

frozen_final_matrix_628hc3a3a2 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

print("\n--- POPULATION CONTRACT ---")

print(
    "Scenarios                  :",
    n_scenarios_628hc3a3a2,
)

print(
    "DEV windows/scenario       :",
    n_dev_628hc3a3a2,
)

print(
    "Clean DEV window shape     :",
    X_DEV_FD002_628D.shape,
)

print(
    "Frozen scalar quality shape:",
    frozen_final_matrix_628hc3a3a2.shape,
)


if n_scenarios_628hc3a3a2 != 315:
    raise RuntimeError(
        "Expected exactly 315 corruption scenarios."
    )

if n_dev_628hc3a3a2 != 8367:
    raise RuntimeError(
        "Expected exactly 8367 DEV windows."
    )

if frozen_final_matrix_628hc3a3a2.shape != (
    315,
    8367,
):
    raise RuntimeError(
        "Frozen C2-C quality-risk matrix "
        "has unexpected shape."
    )


# =============================================================================
# 3. MANIFEST ORDER CONTRACT
# =============================================================================

expected_numbers_628hc3a3a2 = np.arange(
    1,
    316,
    dtype=int,
)

observed_numbers_628hc3a3a2 = (
    manifest_628hc3a3a2[
        "scenario_number"
    ]
    .to_numpy(
        dtype=int
    )
)

scenario_number_exact_628hc3a3a2 = (
    np.array_equal(
        observed_numbers_628hc3a3a2,
        expected_numbers_628hc3a3a2,
    )
)

expected_ids_628hc3a3a2 = np.asarray(
    [
        f"FD002_SCENARIO_{i:03d}"
        for i in range(
            1,
            316,
        )
    ],
    dtype=object,
)

observed_ids_628hc3a3a2 = (
    manifest_628hc3a3a2[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)

scenario_id_exact_628hc3a3a2 = (
    np.array_equal(
        observed_ids_628hc3a3a2,
        expected_ids_628hc3a3a2,
    )
)

print("\n--- MANIFEST ORDER CONTRACT ---")

print(
    "Scenario numbers 001–315 exact:",
    scenario_number_exact_628hc3a3a2,
)

print(
    "Scenario IDs exact             :",
    scenario_id_exact_628hc3a3a2,
)

if not (
    scenario_number_exact_628hc3a3a2
    and
    scenario_id_exact_628hc3a3a2
):
    raise RuntimeError(
        "Manifest order contract failed."
    )


# =============================================================================
# 4. PROTECT THE FROZEN RAW DEV SOURCE
# =============================================================================

base_before_628hc3a3a2 = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .copy(deep=True)
)


# =============================================================================
# 5. PREALLOCATE FOUR CALIBRATED BRANCH MATRICES
# =============================================================================

branch_names_628hc3a3a2 = (
    "level",
    "noise",
    "dropout",
    "stuck",
)

branch_matrices_628hc3a3a2 = {
    branch: np.empty(
        (
            n_scenarios_628hc3a3a2,
            n_dev_628hc3a3a2,
        ),
        dtype=np.float64,
    )
    for branch
    in branch_names_628hc3a3a2
}


# Per-scenario final-risk reconstruction errors.
final_reconstruction_error_628hc3a3a2 = (
    np.empty(
        n_scenarios_628hc3a3a2,
        dtype=np.float64,
    )
)


# Scenario-level diagnostics.
scenario_rows_628hc3a3a2 = []


# =============================================================================
# 6. RECONSTRUCT ALL 315 SCENARIOS
# =============================================================================

print(
    "\n--- REBUILDING FOUR CALIBRATED "
    "BRANCH MATRICES FOR 315 SCENARIOS ---"
)

start_628hc3a3a2 = time.perf_counter()


for i_628hc3a3a2, row_628hc3a3a2 in (
    manifest_628hc3a3a2.iterrows()
):

    scenario_id_628hc3a3a2 = str(
        row_628hc3a3a2[
            "scenario_id"
        ]
    )

    family_628hc3a3a2 = str(
        row_628hc3a3a2[
            "family"
        ]
    )

    sensor_628hc3a3a2 = str(
        row_628hc3a3a2[
            "sensor"
        ]
    )

    severity_628hc3a3a2 = str(
        row_628hc3a3a2[
            "severity"
        ]
    )


    # -------------------------------------------------------------------------
    # A. Apply the exact frozen corruption operator.
    # -------------------------------------------------------------------------

    corrupted_raw_628hc3a3a2 = (
        apply_scenario_628hc0(
            FD002_DEV_CORRUPTION_BASE_628GC,
            row_628hc3a3a2,
        )
    )


    # -------------------------------------------------------------------------
    # B. Re-normalize ONLY the corrupted sensor using frozen regime statistics.
    # -------------------------------------------------------------------------

    normalized_628hc3a3a2 = (
        normalize_corrupted_sensor_628ha(
            corrupted_raw_628hc3a3a2,
            sensor_628hc3a3a2,
        )
    )


    # -------------------------------------------------------------------------
    # C. Rebuild the frozen DEV windows.
    # -------------------------------------------------------------------------

    X_corrupted_628hc3a3a2 = (
        build_windows_628hc0(
            normalized_628hc3a3a2
        )
    )


    if X_corrupted_628hc3a3a2.shape != (
        X_DEV_FD002_628D.shape
    ):
        raise RuntimeError(
            f"{scenario_id_628hc3a3a2}: "
            "corrupted DEV window shape mismatch. "
            f"Observed "
            f"{X_corrupted_628hc3a3a2.shape}, "
            f"expected "
            f"{X_DEV_FD002_628D.shape}."
        )


    # -------------------------------------------------------------------------
    # D. Reconstruct raw C2 sensor-quality branches.
    # -------------------------------------------------------------------------

    raw_branches_628hc3a3a2 = (
        sensor_quality_branches_628hc2a(
            X_corrupted_628hc3a3a2
        )
    )


    # -------------------------------------------------------------------------
    # E. Run the exact frozen C2-B calibration/final-risk chain.
    #
    # This gives:
    #   calibrated level
    #   calibrated noise
    #   calibrated dropout
    #   calibrated stuck
    #   pre_global
    #   sensor_quality_risk
    # -------------------------------------------------------------------------

    quality_result_628hc3a3a2 = (
        final_sensor_quality_from_raw_628hc2b(
            raw_branches_628hc3a3a2
        )
    )


    # -------------------------------------------------------------------------
    # F. Store the four calibrated branch vectors.
    # -------------------------------------------------------------------------

    for branch_628hc3a3a2 in (
        branch_names_628hc3a3a2
    ):

        values_628hc3a3a2 = np.asarray(
            quality_result_628hc3a3a2[
                branch_628hc3a3a2
            ],
            dtype=np.float64,
        ).reshape(-1)

        if values_628hc3a3a2.shape != (
            n_dev_628hc3a3a2,
        ):
            raise RuntimeError(
                f"{scenario_id_628hc3a3a2}: "
                f"{branch_628hc3a3a2} branch "
                "has unexpected shape."
            )

        if not np.isfinite(
            values_628hc3a3a2
        ).all():
            raise RuntimeError(
                f"{scenario_id_628hc3a3a2}: "
                f"{branch_628hc3a3a2} contains "
                "non-finite values."
            )

        if (
            np.min(
                values_628hc3a3a2
            ) < 0.0
            or
            np.max(
                values_628hc3a3a2
            ) > 1.0
        ):
            raise RuntimeError(
                f"{scenario_id_628hc3a3a2}: "
                f"{branch_628hc3a3a2} falls "
                "outside [0,1]."
            )

        branch_matrices_628hc3a3a2[
            branch_628hc3a3a2
        ][
            i_628hc3a3a2
        ] = values_628hc3a3a2


    # -------------------------------------------------------------------------
    # G. Strongest reconstruction audit:
    #
    # Rebuilt final sensor-quality risk must reproduce the already frozen
    # C2-C scalar sensor-quality matrix.
    # -------------------------------------------------------------------------

    rebuilt_final_628hc3a3a2 = np.asarray(
        quality_result_628hc3a3a2[
            "sensor_quality_risk"
        ],
        dtype=np.float64,
    ).reshape(-1)

    frozen_final_628hc3a3a2 = (
        frozen_final_matrix_628hc3a3a2[
            i_628hc3a3a2
        ]
    )

    max_error_628hc3a3a2 = float(
        np.max(
            np.abs(
                rebuilt_final_628hc3a3a2
                -
                frozen_final_628hc3a3a2
            )
        )
    )

    final_reconstruction_error_628hc3a3a2[
        i_628hc3a3a2
    ] = max_error_628hc3a3a2


    if max_error_628hc3a3a2 > 1e-12:

        raise RuntimeError(
            f"{scenario_id_628hc3a3a2}: "
            "reconstructed final quality risk "
            "does not reproduce frozen C2-C. "
            f"max_abs_error={max_error_628hc3a3a2}"
        )


    # -------------------------------------------------------------------------
    # H. Scenario-level branch summary.
    # -------------------------------------------------------------------------

    scenario_rows_628hc3a3a2.append({
        "scenario_id":
            scenario_id_628hc3a3a2,

        "scenario_number":
            int(
                row_628hc3a3a2[
                    "scenario_number"
                ]
            ),

        "family":
            family_628hc3a3a2,

        "sensor":
            sensor_628hc3a3a2,

        "severity":
            severity_628hc3a3a2,

        "level_median":
            float(
                np.median(
                    branch_matrices_628hc3a3a2[
                        "level"
                    ][i_628hc3a3a2]
                )
            ),

        "noise_median":
            float(
                np.median(
                    branch_matrices_628hc3a3a2[
                        "noise"
                    ][i_628hc3a3a2]
                )
            ),

        "dropout_median":
            float(
                np.median(
                    branch_matrices_628hc3a3a2[
                        "dropout"
                    ][i_628hc3a3a2]
                )
            ),

        "stuck_median":
            float(
                np.median(
                    branch_matrices_628hc3a3a2[
                        "stuck"
                    ][i_628hc3a3a2]
                )
            ),

        "final_quality_median":
            float(
                np.median(
                    rebuilt_final_628hc3a3a2
                )
            ),

        "final_reconstruction_max_error":
            max_error_628hc3a3a2,
    })


    # -------------------------------------------------------------------------
    # Progress output.
    # -------------------------------------------------------------------------

    completed_628hc3a3a2 = (
        i_628hc3a3a2 + 1
    )

    if (
        completed_628hc3a3a2 == 1
        or
        completed_628hc3a3a2 % 15 == 0
        or
        completed_628hc3a3a2
        == n_scenarios_628hc3a3a2
    ):

        elapsed_628hc3a3a2 = (
            time.perf_counter()
            -
            start_628hc3a3a2
        )

        rate_628hc3a3a2 = (
            elapsed_628hc3a3a2
            /
            completed_628hc3a3a2
        )

        remaining_628hc3a3a2 = (
            n_scenarios_628hc3a3a2
            -
            completed_628hc3a3a2
        )

        eta_628hc3a3a2 = (
            remaining_628hc3a3a2
            *
            rate_628hc3a3a2
        )

        print(
            f"[{completed_628hc3a3a2:03d}/"
            f"{n_scenarios_628hc3a3a2}] "
            f"{scenario_id_628hc3a3a2} | "
            f"{family_628hc3a3a2:<7} | "
            f"{sensor_628hc3a3a2:<9} | "
            f"{severity_628hc3a3a2:<6} | "
            f"L={np.median(branch_matrices_628hc3a3a2['level'][i_628hc3a3a2]):.4f} | "
            f"N={np.median(branch_matrices_628hc3a3a2['noise'][i_628hc3a3a2]):.4f} | "
            f"D={np.median(branch_matrices_628hc3a3a2['dropout'][i_628hc3a3a2]):.4f} | "
            f"S={np.median(branch_matrices_628hc3a3a2['stuck'][i_628hc3a3a2]):.4f} | "
            f"recon_err={max_error_628hc3a3a2:.2e} | "
            f"elapsed={elapsed_628hc3a3a2:.1f}s | "
            f"ETA≈{eta_628hc3a3a2 / 60.0:.1f} min"
        )


runtime_628hc3a3a2 = (
    time.perf_counter()
    -
    start_628hc3a3a2
)


# =============================================================================
# 7. PUBLISH RECONSTRUCTED BRANCH MATRICES
#
# Separate aliases below reference the same arrays; they do NOT duplicate the
# full matrices in memory.
# =============================================================================

FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2 = (
    branch_matrices_628hc3a3a2
)

FD002_LEVEL_RISK_MATRIX_628HC3A3A2 = (
    FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2[
        "level"
    ]
)

FD002_NOISE_RISK_MATRIX_628HC3A3A2 = (
    FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2[
        "noise"
    ]
)

FD002_DROPOUT_RISK_MATRIX_628HC3A3A2 = (
    FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2[
        "dropout"
    ]
)

FD002_STUCK_RISK_MATRIX_628HC3A3A2 = (
    FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2[
        "stuck"
    ]
)


FD002_SENSOR_QUALITY_BRANCH_RESPONSE_628HC3A3A2 = (
    pd.DataFrame(
        scenario_rows_628hc3a3a2
    )
)


# =============================================================================
# 8. PROVE THE ORIGINAL RAW DEV SOURCE WAS NOT CHANGED
# =============================================================================

clean_source_unchanged_628hc3a3a2 = (
    FD002_DEV_CORRUPTION_BASE_628GC
    .equals(
        base_before_628hc3a3a2
    )
)


# =============================================================================
# 9. MATRIX CONTRACT AUDIT
# =============================================================================

matrix_audit_rows_628hc3a3a2 = []

for branch_628hc3a3a2 in (
    branch_names_628hc3a3a2
):

    matrix_628hc3a3a2 = (
        FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2[
            branch_628hc3a3a2
        ]
    )

    matrix_audit_rows_628hc3a3a2.append({
        "branch":
            branch_628hc3a3a2,

        "shape":
            matrix_628hc3a3a2.shape,

        "finite":
            bool(
                np.isfinite(
                    matrix_628hc3a3a2
                ).all()
            ),

        "within_0_1":
            bool(
                np.min(
                    matrix_628hc3a3a2
                ) >= 0.0
                and
                np.max(
                    matrix_628hc3a3a2
                ) <= 1.0
            ),

        "minimum":
            float(
                np.min(
                    matrix_628hc3a3a2
                )
            ),

        "median":
            float(
                np.median(
                    matrix_628hc3a3a2
                )
            ),

        "maximum":
            float(
                np.max(
                    matrix_628hc3a3a2
                )
            ),
    })


FD002_SENSOR_QUALITY_BRANCH_MATRIX_AUDIT_628HC3A3A2 = (
    pd.DataFrame(
        matrix_audit_rows_628hc3a3a2
    )
)

print(
    "\n--- FOUR-BRANCH MATRIX AUDIT ---"
)

display(
    FD002_SENSOR_QUALITY_BRANCH_MATRIX_AUDIT_628HC3A3A2
)


# =============================================================================
# 10. FROZEN FINAL-RISK RECONSTRUCTION AUDIT
# =============================================================================

max_final_error_628hc3a3a2 = float(
    np.max(
        final_reconstruction_error_628hc3a3a2
    )
)

exact_final_reconstruction_628hc3a3a2 = (
    max_final_error_628hc3a3a2
    <=
    1e-12
)

print(
    "\n--- FROZEN C2-C FINAL-RISK RECONSTRUCTION AUDIT ---"
)

print(
    "Maximum absolute reconstruction error:",
    max_final_error_628hc3a3a2,
)

print(
    "Exact <= 1e-12                    :",
    exact_final_reconstruction_628hc3a3a2,
)


# =============================================================================
# 11. SCENARIO RESPONSE TABLE
# =============================================================================

print(
    "\n--- FIRST 15 SCENARIO BRANCH RESPONSES ---"
)

display(
    FD002_SENSOR_QUALITY_BRANCH_RESPONSE_628HC3A3A2
    .head(15)
)


# =============================================================================
# 12. FAMILY × SEVERITY BRANCH RESPONSE
# =============================================================================

family_severity_628hc3a3a2 = (
    FD002_SENSOR_QUALITY_BRANCH_RESPONSE_628HC3A3A2
    .groupby(
        [
            "family",
            "severity",
        ],
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "size",
        ),

        median_level=(
            "level_median",
            "median",
        ),

        median_noise=(
            "noise_median",
            "median",
        ),

        median_dropout=(
            "dropout_median",
            "median",
        ),

        median_stuck=(
            "stuck_median",
            "median",
        ),

        median_final_quality=(
            "final_quality_median",
            "median",
        ),
    )
    .reset_index()
)

FD002_SENSOR_QUALITY_BRANCH_FAMILY_SEVERITY_628HC3A3A2 = (
    family_severity_628hc3a3a2
)

print(
    "\n--- FAMILY × SEVERITY FOUR-BRANCH RESPONSE ---"
)

display(
    FD002_SENSOR_QUALITY_BRANCH_FAMILY_SEVERITY_628HC3A3A2
)


# =============================================================================
# 13. FINAL AUDIT
# =============================================================================

all_matrix_shapes_correct_628hc3a3a2 = bool(
    all(
        matrix.shape
        ==
        (
            315,
            8367,
        )

        for matrix in (
            FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2
            .values()
        )
    )
)

all_matrix_values_finite_628hc3a3a2 = bool(
    all(
        np.isfinite(
            matrix
        ).all()

        for matrix in (
            FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2
            .values()
        )
    )
)

all_matrix_values_within_0_1_628hc3a3a2 = bool(
    all(
        (
            np.min(matrix) >= 0.0
            and
            np.max(matrix) <= 1.0
        )

        for matrix in (
            FD002_SENSOR_QUALITY_BRANCH_MATRICES_628HC3A3A2
            .values()
        )
    )
)


audit_628hc3a3a2 = pd.DataFrame([
    {
        "check":
            "scenario_count_315",

        "passed":
            n_scenarios_628hc3a3a2
            ==
            315,
    },

    {
        "check":
            "dev_windows_8367",

        "passed":
            n_dev_628hc3a3a2
            ==
            8367,
    },

    {
        "check":
            "manifest_order_exact",

        "passed":
            scenario_number_exact_628hc3a3a2
            and
            scenario_id_exact_628hc3a3a2,
    },

    {
        "check":
            "four_matrix_shapes_exact",

        "passed":
            all_matrix_shapes_correct_628hc3a3a2,
    },

    {
        "check":
            "four_matrices_finite",

        "passed":
            all_matrix_values_finite_628hc3a3a2,
    },

    {
        "check":
            "four_matrices_within_0_1",

        "passed":
            all_matrix_values_within_0_1_628hc3a3a2,
    },

    {
        "check":
            "frozen_final_risk_reproduced",

        "passed":
            exact_final_reconstruction_628hc3a3a2,
    },

    {
        "check":
            "clean_raw_source_unchanged",

        "passed":
            clean_source_unchanged_628hc3a3a2,
    },
])


print(
    "\n--- FINAL C3-A3-A2 AUDIT ---"
)

display(
    audit_628hc3a3a2
)


all_checks_628hc3a3a2 = bool(
    audit_628hc3a3a2[
        "passed"
    ].all()
)


# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print(
    "STEP 6.28H-C3-A3-A2 FINAL SUMMARY"
)
print("=" * 100)

print(
    "Scenarios reconstructed             :",
    n_scenarios_628hc3a3a2,
)

print(
    "DEV windows/scenario                :",
    n_dev_628hc3a3a2,
)

print(
    "Level-risk matrix shape             :",
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2.shape,
)

print(
    "Noise-risk matrix shape             :",
    FD002_NOISE_RISK_MATRIX_628HC3A3A2.shape,
)

print(
    "Dropout-risk matrix shape           :",
    FD002_DROPOUT_RISK_MATRIX_628HC3A3A2.shape,
)

print(
    "Stuck-risk matrix shape             :",
    FD002_STUCK_RISK_MATRIX_628HC3A3A2.shape,
)

print(
    "Frozen scalar quality reproduced    :",
    exact_final_reconstruction_628hc3a3a2,
)

print(
    "Maximum reconstruction error        :",
    max_final_error_628hc3a3a2,
)

print(
    "Frozen raw DEV source unchanged     :",
    clean_source_unchanged_628hc3a3a2,
)

print(
    "All C3-A3-A2 checks passed          :",
    all_checks_628hc3a3a2,
)

print(
    "Runtime (sec)                       :",
    runtime_628hc3a3a2,
)

print("DEV RUL labels used                  : False")
print("Official TEST labels read            : False")
print("TEST scored                          : False")
print("Adaptive fusion rule selected        : False")
print("Adaptive threshold selected          : False")
print("Model executed                       : False")
print("Model retrained                      : False")
print("Files written                        : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A2 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0
# LOCATE RECONSTRUCTED FOUR-BRANCH MATRICES BEFORE FREEZING
# ================================================================

import numpy as np
import pandas as pd

print("=" * 96)
print("STEP 6.28H-C3-A3-A2-F0 — LOCATE FOUR-BRANCH MATRICES FOR FREEZE")
print("=" * 96)

EXPECTED_SHAPE_628HC3A3A2F0 = (315, 8367)

# ------------------------------------------------
# 1. Locate direct ndarray candidates
# ------------------------------------------------
candidate_rows_628hc3a3a2f0 = []

for name, obj in globals().items():

    if isinstance(obj, np.ndarray) and obj.shape == EXPECTED_SHAPE_628HC3A3A2F0:

        lname = name.lower()

        branch_hits = [
            b for b in ("level", "noise", "dropout", "stuck")
            if b in lname
        ]

        if branch_hits:
            candidate_rows_628hc3a3a2f0.append(
                {
                    "object_name": name,
                    "type": type(obj).__name__,
                    "shape": obj.shape,
                    "finite": bool(np.isfinite(obj).all()),
                    "minimum": float(np.min(obj)),
                    "median": float(np.median(obj)),
                    "maximum": float(np.max(obj)),
                    "branch_name_hits": ",".join(branch_hits),
                }
            )

direct_candidates_628hc3a3a2f0 = pd.DataFrame(
    candidate_rows_628hc3a3a2f0
)

print("\n--- DIRECT FOUR-BRANCH MATRIX CANDIDATES ---")

if len(direct_candidates_628hc3a3a2f0) == 0:
    print("None found.")
else:
    display(
        direct_candidates_628hc3a3a2f0.sort_values(
            ["branch_name_hits", "object_name"]
        ).reset_index(drop=True)
    )


# ------------------------------------------------
# 2. Locate dictionary containers holding
#    four matrices of expected shape
# ------------------------------------------------
dict_rows_628hc3a3a2f0 = []

required_branches_628hc3a3a2f0 = {
    "level",
    "noise",
    "dropout",
    "stuck",
}

for name, obj in globals().items():

    if not isinstance(obj, dict):
        continue

    keys_lower = {str(k).lower() for k in obj.keys()}

    if not required_branches_628hc3a3a2f0.issubset(keys_lower):
        continue

    branch_info = {}
    valid_container = True

    for branch in sorted(required_branches_628hc3a3a2f0):

        # recover original key case safely
        matching_keys = [
            k for k in obj.keys()
            if str(k).lower() == branch
        ]

        if len(matching_keys) != 1:
            valid_container = False
            break

        arr = np.asarray(obj[matching_keys[0]])

        branch_info[branch] = {
            "shape": arr.shape,
            "finite": (
                bool(np.isfinite(arr).all())
                if np.issubdtype(arr.dtype, np.number)
                else False
            ),
        }

        if arr.shape != EXPECTED_SHAPE_628HC3A3A2F0:
            valid_container = False

    if valid_container:
        dict_rows_628hc3a3a2f0.append(
            {
                "object_name": name,
                "type": type(obj).__name__,
                "keys": list(obj.keys()),
                "level_shape": branch_info["level"]["shape"],
                "noise_shape": branch_info["noise"]["shape"],
                "dropout_shape": branch_info["dropout"]["shape"],
                "stuck_shape": branch_info["stuck"]["shape"],
                "all_finite": all(
                    branch_info[b]["finite"]
                    for b in required_branches_628hc3a3a2f0
                ),
            }
        )

dict_candidates_628hc3a3a2f0 = pd.DataFrame(
    dict_rows_628hc3a3a2f0
)

print("\n--- FOUR-BRANCH DICTIONARY CANDIDATES ---")

if len(dict_candidates_628hc3a3a2f0) == 0:
    print("None found.")
else:
    display(dict_candidates_628hc3a3a2f0)


# ------------------------------------------------
# 3. Safety statement
# ------------------------------------------------
print("\n--- FREEZE PRE-FLIGHT SAFETY STATUS ---")
print("Expected matrix shape :", EXPECTED_SHAPE_628HC3A3A2F0)
print("Files written         : False")
print("DEV RUL labels used   : False")
print("TEST labels read      : False")
print("TEST scored           : False")
print("Threshold selected    : False")
print("Model executed        : False")
print("Model retrained       : False")

print("\n=== END STEP 6.28H-C3-A3-A2-F0 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0-R1
# EXACT FOUR-BRANCH FREEZE PRE-FLIGHT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 96)
print("STEP 6.28H-C3-A3-A2-F0-R1 — EXACT FOUR-BRANCH FREEZE PRE-FLIGHT")
print("=" * 96)

expected_shape_628hc3a3a2f0r1 = (315, 8367)

branch_objects_628hc3a3a2f0r1 = {
    "level": FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    "noise": FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    "dropout": FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
    "stuck": FD002_STUCK_RISK_MATRIX_628HC3A3A2,
}

rows_628hc3a3a2f0r1 = []

for branch, arr in branch_objects_628hc3a3a2f0r1.items():

    arr = np.asarray(arr, dtype=np.float64)

    rows_628hc3a3a2f0r1.append(
        {
            "branch": branch,
            "shape": arr.shape,
            "shape_exact": arr.shape == expected_shape_628hc3a3a2f0r1,
            "finite": bool(np.isfinite(arr).all()),
            "within_0_1": bool(
                np.all((arr >= 0.0) & (arr <= 1.0))
            ),
            "minimum": float(np.min(arr)),
            "median": float(np.median(arr)),
            "maximum": float(np.max(arr)),
        }
    )

freeze_preflight_628hc3a3a2f0r1 = pd.DataFrame(
    rows_628hc3a3a2f0r1
)

print("\n--- FOUR-BRANCH EXACT CONTRACT ---")
display(freeze_preflight_628hc3a3a2f0r1)

all_shape_exact_628hc3a3a2f0r1 = bool(
    freeze_preflight_628hc3a3a2f0r1["shape_exact"].all()
)

all_finite_628hc3a3a2f0r1 = bool(
    freeze_preflight_628hc3a3a2f0r1["finite"].all()
)

all_within_0_1_628hc3a3a2f0r1 = bool(
    freeze_preflight_628hc3a3a2f0r1["within_0_1"].all()
)


# ------------------------------------------------
# Cross-check against the already validated
# frozen scalar sensor-quality risk matrix
# ------------------------------------------------
scalar_reconstructed_628hc3a3a2f0r1 = (
    final_sensor_quality_from_raw_628hc2b(
        branch_objects_628hc3a3a2f0r1
    )["sensor_quality_risk"]
)

scalar_reconstructed_628hc3a3a2f0r1 = np.asarray(
    scalar_reconstructed_628hc3a3a2f0r1,
    dtype=np.float64
)

scalar_frozen_628hc3a3a2f0r1 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

scalar_max_abs_error_628hc3a3a2f0r1 = float(
    np.max(
        np.abs(
            scalar_reconstructed_628hc3a3a2f0r1
            - scalar_frozen_628hc3a3a2f0r1
        )
    )
)

scalar_exact_628hc3a3a2f0r1 = (
    scalar_max_abs_error_628hc3a3a2f0r1 <= 1e-12
)

print("\n--- FROZEN SCALAR-RISK REPRODUCTION ---")
print(
    "Maximum absolute error :",
    scalar_max_abs_error_628hc3a3a2f0r1
)
print(
    "Exact <= 1e-12         :",
    scalar_exact_628hc3a3a2f0r1
)


# ------------------------------------------------
# Final pre-flight audit
# ------------------------------------------------
audit_628hc3a3a2f0r1 = pd.DataFrame(
    {
        "check": [
            "four_shapes_exact",
            "four_matrices_finite",
            "four_matrices_within_0_1",
            "frozen_scalar_risk_reproduced",
        ],
        "passed": [
            all_shape_exact_628hc3a3a2f0r1,
            all_finite_628hc3a3a2f0r1,
            all_within_0_1_628hc3a3a2f0r1,
            scalar_exact_628hc3a3a2f0r1,
        ],
    }
)

print("\n--- FREEZE PRE-FLIGHT AUDIT ---")
display(audit_628hc3a3a2f0r1)

all_checks_passed_628hc3a3a2f0r1 = bool(
    audit_628hc3a3a2f0r1["passed"].all()
)

print("\n" + "=" * 96)
print("STEP 6.28H-C3-A3-A2-F0-R1 FINAL SUMMARY")
print("=" * 96)

print(
    "Level matrix shape             :",
    np.asarray(
        FD002_LEVEL_RISK_MATRIX_628HC3A3A2
    ).shape
)
print(
    "Noise matrix shape             :",
    np.asarray(
        FD002_NOISE_RISK_MATRIX_628HC3A3A2
    ).shape
)
print(
    "Dropout matrix shape           :",
    np.asarray(
        FD002_DROPOUT_RISK_MATRIX_628HC3A3A2
    ).shape
)
print(
    "Stuck matrix shape             :",
    np.asarray(
        FD002_STUCK_RISK_MATRIX_628HC3A3A2
    ).shape
)

print(
    "Frozen scalar risk reproduced  :",
    scalar_exact_628hc3a3a2f0r1
)
print(
    "All freeze pre-flight checks   :",
    all_checks_passed_628hc3a3a2f0r1
)

print("DEV RUL labels used            : False")
print("Official TEST labels read      : False")
print("TEST scored                    : False")
print("Threshold selected             : False")
print("Adaptive rule selected         : False")
print("Model executed                 : False")
print("Model retrained                : False")
print("Files written                  : False")

if not all_checks_passed_628hc3a3a2f0r1:
    raise RuntimeError(
        "Four-branch freeze pre-flight failed. "
        "Do not write frozen artifacts."
    )

print("\n=== END STEP 6.28H-C3-A3-A2-F0-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0-R2
# EXACT SCENARIO-WISE SCALAR-RISK REPRODUCTION BEFORE FREEZE
# ================================================================

import numpy as np
import pandas as pd
import time

print("=" * 100)
print(
    "STEP 6.28H-C3-A3-A2-F0-R2 — "
    "EXACT SCENARIO-WISE SCALAR-RISK REPRODUCTION"
)
print("=" * 100)

# ---------------------------------------------------------------
# 1. Exact frozen four-branch matrices
# ---------------------------------------------------------------

branch_matrices_628hc3a3a2f0r2 = {
    "level": np.asarray(
        FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64,
    ),
    "noise": np.asarray(
        FD002_NOISE_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64,
    ),
    "dropout": np.asarray(
        FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64,
    ),
    "stuck": np.asarray(
        FD002_STUCK_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64,
    ),
}

expected_shape_628hc3a3a2f0r2 = (315, 8367)

for branch, matrix in branch_matrices_628hc3a3a2f0r2.items():
    if matrix.shape != expected_shape_628hc3a3a2f0r2:
        raise RuntimeError(
            f"{branch} matrix has unexpected shape {matrix.shape}."
        )

print("\nFour branch matrices confirmed:", expected_shape_628hc3a3a2f0r2)


# ---------------------------------------------------------------
# 2. Reconstruct final sensor-quality risk SCENARIO BY SCENARIO
# ---------------------------------------------------------------

print(
    "\n--- RECONSTRUCTING FINAL SENSOR-QUALITY RISK "
    "SCENARIO BY SCENARIO ---"
)

t0_628hc3a3a2f0r2 = time.perf_counter()

scalar_rows_628hc3a3a2f0r2 = []

for i in range(315):

    scenario_branch_dict_628hc3a3a2f0r2 = {
        "level":
            branch_matrices_628hc3a3a2f0r2["level"][i],
        "noise":
            branch_matrices_628hc3a3a2f0r2["noise"][i],
        "dropout":
            branch_matrices_628hc3a3a2f0r2["dropout"][i],
        "stuck":
            branch_matrices_628hc3a3a2f0r2["stuck"][i],
    }

    result_628hc3a3a2f0r2 = (
        final_sensor_quality_from_raw_628hc2b(
            scenario_branch_dict_628hc3a3a2f0r2
        )
    )

    row_risk_628hc3a3a2f0r2 = np.asarray(
        result_628hc3a3a2f0r2["sensor_quality_risk"],
        dtype=np.float64,
    ).reshape(-1)

    if row_risk_628hc3a3a2f0r2.shape != (8367,):
        raise RuntimeError(
            f"Scenario {i + 1}: unexpected scalar-risk "
            f"shape {row_risk_628hc3a3a2f0r2.shape}."
        )

    scalar_rows_628hc3a3a2f0r2.append(
        row_risk_628hc3a3a2f0r2
    )

    if (
        (i + 1) == 1
        or (i + 1) % 50 == 0
        or (i + 1) == 315
    ):
        print(
            f"[{i + 1:03d}/315] "
            f"reconstructed"
        )

scalar_reconstructed_628hc3a3a2f0r2 = np.vstack(
    scalar_rows_628hc3a3a2f0r2
)

runtime_628hc3a3a2f0r2 = (
    time.perf_counter() - t0_628hc3a3a2f0r2
)

print(
    "\nReconstructed scalar-risk matrix shape:",
    scalar_reconstructed_628hc3a3a2f0r2.shape,
)


# ---------------------------------------------------------------
# 3. Compare against frozen C2-C scalar-risk matrix
# ---------------------------------------------------------------

scalar_frozen_628hc3a3a2f0r2 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

print(
    "Frozen C2-C scalar-risk matrix shape :",
    scalar_frozen_628hc3a3a2f0r2.shape,
)

if (
    scalar_reconstructed_628hc3a3a2f0r2.shape
    != scalar_frozen_628hc3a3a2f0r2.shape
):
    raise RuntimeError(
        "Reconstructed and frozen scalar-risk shapes differ."
    )

scalar_abs_diff_628hc3a3a2f0r2 = np.abs(
    scalar_reconstructed_628hc3a3a2f0r2
    - scalar_frozen_628hc3a3a2f0r2
)

scalar_max_error_628hc3a3a2f0r2 = float(
    np.max(scalar_abs_diff_628hc3a3a2f0r2)
)

scalar_mean_error_628hc3a3a2f0r2 = float(
    np.mean(scalar_abs_diff_628hc3a3a2f0r2)
)

scalar_exact_628hc3a3a2f0r2 = bool(
    scalar_max_error_628hc3a3a2f0r2 <= 1e-12
)

print("\n--- FROZEN C2-C REPRODUCTION AUDIT ---")
print(
    "Maximum absolute error :",
    scalar_max_error_628hc3a3a2f0r2,
)
print(
    "Mean absolute error    :",
    scalar_mean_error_628hc3a3a2f0r2,
)
print(
    "Exact <= 1e-12         :",
    scalar_exact_628hc3a3a2f0r2,
)


# ---------------------------------------------------------------
# 4. Final freeze pre-flight audit
# ---------------------------------------------------------------

audit_628hc3a3a2f0r2 = pd.DataFrame(
    {
        "check": [
            "level_shape_exact",
            "noise_shape_exact",
            "dropout_shape_exact",
            "stuck_shape_exact",
            "all_branch_values_finite",
            "all_branch_values_within_0_1",
            "scalar_matrix_shape_exact",
            "frozen_C2C_scalar_risk_reproduced",
        ],
        "passed": [
            branch_matrices_628hc3a3a2f0r2["level"].shape
            == expected_shape_628hc3a3a2f0r2,

            branch_matrices_628hc3a3a2f0r2["noise"].shape
            == expected_shape_628hc3a3a2f0r2,

            branch_matrices_628hc3a3a2f0r2["dropout"].shape
            == expected_shape_628hc3a3a2f0r2,

            branch_matrices_628hc3a3a2f0r2["stuck"].shape
            == expected_shape_628hc3a3a2f0r2,

            all(
                np.isfinite(x).all()
                for x in branch_matrices_628hc3a3a2f0r2.values()
            ),

            all(
                np.all((x >= 0.0) & (x <= 1.0))
                for x in branch_matrices_628hc3a3a2f0r2.values()
            ),

            scalar_reconstructed_628hc3a3a2f0r2.shape
            == (315, 8367),

            scalar_exact_628hc3a3a2f0r2,
        ],
    }
)

print("\n--- FINAL FREEZE PRE-FLIGHT AUDIT ---")
display(audit_628hc3a3a2f0r2)

all_checks_passed_628hc3a3a2f0r2 = bool(
    audit_628hc3a3a2f0r2["passed"].all()
)


# ---------------------------------------------------------------
# 5. Summary
# ---------------------------------------------------------------

print("\n" + "=" * 100)
print("STEP 6.28H-C3-A3-A2-F0-R2 FINAL SUMMARY")
print("=" * 100)

print(
    "Four branch matrix shape            :",
    expected_shape_628hc3a3a2f0r2,
)
print(
    "Reconstructed scalar-risk shape     :",
    scalar_reconstructed_628hc3a3a2f0r2.shape,
)
print(
    "Frozen scalar-risk shape            :",
    scalar_frozen_628hc3a3a2f0r2.shape,
)
print(
    "Maximum reconstruction error        :",
    scalar_max_error_628hc3a3a2f0r2,
)
print(
    "Frozen C2-C scalar risk reproduced  :",
    scalar_exact_628hc3a3a2f0r2,
)
print(
    "All freeze pre-flight checks passed :",
    all_checks_passed_628hc3a3a2f0r2,
)
print(
    "Runtime (sec)                       :",
    runtime_628hc3a3a2f0r2,
)

print("DEV RUL labels used                  : False")
print("Official TEST labels read            : False")
print("TEST scored                          : False")
print("Threshold selected                   : False")
print("Adaptive fusion rule selected        : False")
print("Model executed                       : False")
print("Model retrained                      : False")
print("Files written                        : False")

if not all_checks_passed_628hc3a3a2f0r2:
    raise RuntimeError(
        "Freeze pre-flight did not pass. "
        "Do not write frozen artifacts."
    )

print("\n=== END STEP 6.28H-C3-A3-A2-F0-R2 ===")

In [ ]:
    # ================================================================
# STEP 6.28H-C3-A3-A2-F0-R3
# CORRECT FOUR-BRANCH FREEZE READINESS AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print(
    "STEP 6.28H-C3-A3-A2-F0-R3 — "
    "CORRECT FOUR-BRANCH FREEZE READINESS AUDIT"
)
print("=" * 100)

expected_shape = (315, 8367)

branch_objects = {
    "level": np.asarray(
        FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64
    ),
    "noise": np.asarray(
        FD002_NOISE_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64
    ),
    "dropout": np.asarray(
        FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64
    ),
    "stuck": np.asarray(
        FD002_STUCK_RISK_MATRIX_628HC3A3A2,
        dtype=np.float64
    ),
}

frozen_scalar = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

rows = []

for name, arr in branch_objects.items():
    rows.append(
        {
            "object": name,
            "shape": arr.shape,
            "shape_exact": arr.shape == expected_shape,
            "finite": bool(np.isfinite(arr).all()),
            "within_0_1": bool(
                np.all((arr >= 0.0) & (arr <= 1.0))
            ),
            "minimum": float(np.min(arr)),
            "median": float(np.median(arr)),
            "maximum": float(np.max(arr)),
        }
    )

audit_df = pd.DataFrame(rows)

print("\n--- FOUR CALIBRATED BRANCH MATRICES ---")
display(audit_df)

print("\n--- FROZEN SCALAR SENSOR-QUALITY MATRIX ---")
print("Shape       :", frozen_scalar.shape)
print("Finite      :", bool(np.isfinite(frozen_scalar).all()))
print(
    "Within [0,1]:",
    bool(np.all((frozen_scalar >= 0.0) & (frozen_scalar <= 1.0)))
)

branch_contract_passed = bool(
    audit_df["shape_exact"].all()
    and audit_df["finite"].all()
    and audit_df["within_0_1"].all()
)

scalar_contract_passed = bool(
    frozen_scalar.shape == expected_shape
    and np.isfinite(frozen_scalar).all()
    and np.all(
        (frozen_scalar >= 0.0)
        & (frozen_scalar <= 1.0)
    )
)

print("\n--- PIPELINE CONTRACT STATUS ---")
print(
    "Four calibrated branch matrices valid :",
    branch_contract_passed
)
print(
    "Frozen C2-C scalar matrix valid        :",
    scalar_contract_passed
)

print(
    "\nImportant:"
)
print(
    "The four matrices above are CALIBRATED branch risks."
)
print(
    "final_sensor_quality_from_raw_628hc2b() expects RAW branch "
    "scores and must not be applied to these matrices again."
)
print(
    "Exact branch-to-final reconstruction was already established "
    "inside STEP 6.28H-C3-A3-A2 with max error = 0.0."
)

all_ready = bool(
    branch_contract_passed
    and scalar_contract_passed
)

print("\n" + "=" * 100)
print("FREEZE READINESS SUMMARY")
print("=" * 100)
print("Branch contract passed       :", branch_contract_passed)
print("Scalar contract passed       :", scalar_contract_passed)
print("Ready for four-branch freeze :", all_ready)
print("DEV RUL labels used          : False")
print("Official TEST labels read    : False")
print("TEST scored                  : False")
print("Threshold selected           : False")
print("Model executed               : False")
print("Model retrained              : False")
print("Files written                : False")

if not all_ready:
    raise RuntimeError(
        "Freeze readiness contract failed."
    )

print("\n=== END STEP 6.28H-C3-A3-A2-F0-R3 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0-R3
# FOUR-BRANCH FREEZE READINESS AUDIT — NO LOOP VERSION
# ================================================================

import numpy as np

print("=" * 96)
print("STEP 6.28H-C3-A3-A2-F0-R3 — FOUR-BRANCH FREEZE READINESS AUDIT")
print("=" * 96)

expected_shape = (315, 8367)

level = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64
)

noise = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64
)

dropout = np.asarray(
    FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64
)

stuck = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64
)

scalar = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64
)

level_ok = (
    level.shape == expected_shape
    and np.isfinite(level).all()
    and np.all((level >= 0.0) & (level <= 1.0))
)

noise_ok = (
    noise.shape == expected_shape
    and np.isfinite(noise).all()
    and np.all((noise >= 0.0) & (noise <= 1.0))
)

dropout_ok = (
    dropout.shape == expected_shape
    and np.isfinite(dropout).all()
    and np.all((dropout >= 0.0) & (dropout <= 1.0))
)

stuck_ok = (
    stuck.shape == expected_shape
    and np.isfinite(stuck).all()
    and np.all((stuck >= 0.0) & (stuck <= 1.0))
)

scalar_ok = (
    scalar.shape == expected_shape
    and np.isfinite(scalar).all()
    and np.all((scalar >= 0.0) & (scalar <= 1.0))
)

print("\n--- MATRIX CONTRACT ---")
print("level   :", level.shape,   "| valid =", bool(level_ok))
print("noise   :", noise.shape,   "| valid =", bool(noise_ok))
print("dropout :", dropout.shape, "| valid =", bool(dropout_ok))
print("stuck   :", stuck.shape,   "| valid =", bool(stuck_ok))
print("scalar  :", scalar.shape,  "| valid =", bool(scalar_ok))

print("\n--- RANGE DIAGNOSTIC ---")
print(
    "level   min/median/max:",
    float(level.min()),
    float(np.median(level)),
    float(level.max())
)

print(
    "noise   min/median/max:",
    float(noise.min()),
    float(np.median(noise)),
    float(noise.max())
)

print(
    "dropout min/median/max:",
    float(dropout.min()),
    float(np.median(dropout)),
    float(dropout.max())
)

print(
    "stuck   min/median/max:",
    float(stuck.min()),
    float(np.median(stuck)),
    float(stuck.max())
)

print(
    "scalar  min/median/max:",
    float(scalar.min()),
    float(np.median(scalar)),
    float(scalar.max())
)

branch_contract_passed = bool(
    level_ok
    and noise_ok
    and dropout_ok
    and stuck_ok
)

all_ready = bool(
    branch_contract_passed
    and scalar_ok
)

print("\n--- FREEZE READINESS SUMMARY ---")
print(
    "Branch contract passed       :",
    branch_contract_passed
)

print(
    "Scalar contract passed       :",
    bool(scalar_ok)
)

print(
    "Ready for four-branch freeze :",
    all_ready
)

print("\n--- SAFETY CONTRACT ---")
print("DEV RUL labels used          : False")
print("Official TEST labels read    : False")
print("TEST scored                  : False")
print("Threshold selected           : False")
print("Adaptive fusion selected     : False")
print("Model executed               : False")
print("Model retrained              : False")
print("Files written                : False")

if not all_ready:
    raise RuntimeError(
        "Freeze readiness contract failed. "
        "Do not write frozen artifacts."
    )

print("\nR3 audit passed. No files were written.")
print(
    "=== END STEP 6.28H-C3-A3-A2-F0-R3 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0-R4A
# SENSOR-QUALITY COMBINATION LINEAGE INSPECTION
# ================================================================

import inspect
import dis

print("=" * 94)
print("STEP 6.28H-C3-A3-A2-F0-R4A — SENSOR-QUALITY COMBINATION LINEAGE")
print("=" * 94)

functions_to_check = {
    "calibrate_branch_scores_628hc2b":
        calibrate_branch_scores_628hc2b,

    "final_sensor_quality_from_raw_628hc2b":
        final_sensor_quality_from_raw_628hc2b,
}

for name, func in functions_to_check.items():

    print("\n" + "-" * 94)
    print(name)
    print("-" * 94)

    print("Signature:", inspect.signature(func))

    try:
        source = inspect.getsource(func)
        print("\nSOURCE CODE:\n")
        print(source)

    except (OSError, TypeError):
        print(
            "\nSource text could not be recovered directly."
        )
        print("Showing Python bytecode instead:\n")
        dis.dis(func)

print("\n--- SAFETY CONTRACT ---")
print("DEV RUL labels used       : False")
print("Official TEST labels read : False")
print("TEST scored               : False")
print("Threshold selected        : False")
print("Adaptive fusion selected  : False")
print("Model executed            : False")
print("Model retrained           : False")
print("Files written             : False")

print("\n=== END STEP 6.28H-C3-A3-A2-F0-R4A ===")

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F0-R4B
# EXACT SCALAR-RISK RECONSTRUCTION FROM CALIBRATED BRANCH MATRICES
# ================================================================

import numpy as np
import pandas as pd

print("=" * 104)
print(
    "STEP 6.28H-C3-A3-A2-F0-R4B — "
    "EXACT SCALAR-RISK RECONSTRUCTION FROM CALIBRATED BRANCHES"
)
print("=" * 104)


# ------------------------------------------------
# 1. LOAD THE FOUR ALREADY-CALIBRATED BRANCH MATRICES
# ------------------------------------------------

level_628hc3a3a2r4b = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

noise_628hc3a3a2r4b = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

dropout_628hc3a3a2r4b = np.asarray(
    FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3a2r4b = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

frozen_scalar_628hc3a3a2r4b = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)


# ------------------------------------------------
# 2. CONTRACT
# ------------------------------------------------

expected_shape_628hc3a3a2r4b = (315, 8367)

branch_matrices_628hc3a3a2r4b = {
    "level": level_628hc3a3a2r4b,
    "noise": noise_628hc3a3a2r4b,
    "dropout": dropout_628hc3a3a2r4b,
    "stuck": stuck_628hc3a3a2r4b,
}

print("\n--- CALIBRATED BRANCH CONTRACT ---")

contract_rows_628hc3a3a2r4b = []

for name, matrix in branch_matrices_628hc3a3a2r4b.items():

    shape_exact = (
        matrix.shape
        == expected_shape_628hc3a3a2r4b
    )

    finite = bool(
        np.isfinite(matrix).all()
    )

    within_01 = bool(
        np.min(matrix) >= 0.0
        and np.max(matrix) <= 1.0
    )

    contract_rows_628hc3a3a2r4b.append(
        {
            "branch": name,
            "shape": matrix.shape,
            "shape_exact": shape_exact,
            "finite": finite,
            "within_0_1": within_01,
        }
    )

display(
    pd.DataFrame(
        contract_rows_628hc3a3a2r4b
    )
)


# ------------------------------------------------
# 3. PRE-GLOBAL RISK
#
# IMPORTANT:
# These branch matrices are ALREADY calibrated.
# Therefore DO NOT call calibrate_branch_scores_628hc2b().
# ------------------------------------------------

pre_global_matrix_628hc3a3a2r4b = (
    np.maximum.reduce(
        [
            level_628hc3a3a2r4b,
            noise_628hc3a3a2r4b,
            dropout_628hc3a3a2r4b,
            stuck_628hc3a3a2r4b,
        ]
    )
)

print("\n--- PRE-GLOBAL MATRIX ---")
print(
    "Shape :",
    pre_global_matrix_628hc3a3a2r4b.shape,
)
print(
    "Finite:",
    bool(
        np.isfinite(
            pre_global_matrix_628hc3a3a2r4b
        ).all()
    ),
)
print(
    "Range :",
    float(
        np.min(
            pre_global_matrix_628hc3a3a2r4b
        )
    ),
    "to",
    float(
        np.max(
            pre_global_matrix_628hc3a3a2r4b
        )
    ),
)


# ------------------------------------------------
# 4. APPLY ONLY THE FROZEN GLOBAL CALIBRATION
#
# empirical_midrank_1d works on 1-D values.
# We therefore process each scenario row independently.
# ------------------------------------------------

reconstructed_scalar_628hc3a3a2r4b = (
    np.empty(
        expected_shape_628hc3a3a2r4b,
        dtype=np.float64,
    )
)

print(
    "\n--- APPLYING FROZEN GLOBAL CALIBRATION "
    "SCENARIO BY SCENARIO ---"
)

for i in range(
    expected_shape_628hc3a3a2r4b[0]
):

    reconstructed_scalar_628hc3a3a2r4b[i] = (
        empirical_midrank_1d_628hc2b(
            pre_global_matrix_628hc3a3a2r4b[i],
            FIT_GLOBAL_REFERENCE_628HC2B,
        )
    )

    if (
        i == 0
        or (i + 1) % 50 == 0
        or i + 1
        == expected_shape_628hc3a3a2r4b[0]
    ):
        print(
            f"[{i + 1:03d}/315] reconstructed"
        )


# ------------------------------------------------
# 5. EXACT REPRODUCTION AUDIT
# ------------------------------------------------

difference_628hc3a3a2r4b = (
    reconstructed_scalar_628hc3a3a2r4b
    - frozen_scalar_628hc3a3a2r4b
)

max_abs_error_628hc3a3a2r4b = float(
    np.max(
        np.abs(
            difference_628hc3a3a2r4b
        )
    )
)

mean_abs_error_628hc3a3a2r4b = float(
    np.mean(
        np.abs(
            difference_628hc3a3a2r4b
        )
    )
)

exact_1e12_628hc3a3a2r4b = bool(
    max_abs_error_628hc3a3a2r4b
    <= 1e-12
)

print(
    "\n--- FROZEN C2-C SCALAR-RISK "
    "REPRODUCTION AUDIT ---"
)

print(
    "Reconstructed shape       :",
    reconstructed_scalar_628hc3a3a2r4b.shape,
)

print(
    "Frozen scalar shape       :",
    frozen_scalar_628hc3a3a2r4b.shape,
)

print(
    "Maximum absolute error    :",
    max_abs_error_628hc3a3a2r4b,
)

print(
    "Mean absolute error       :",
    mean_abs_error_628hc3a3a2r4b,
)

print(
    "Exact <= 1e-12            :",
    exact_1e12_628hc3a3a2r4b,
)


# ------------------------------------------------
# 6. FINAL AUDIT
# ------------------------------------------------

audit_628hc3a3a2r4b = pd.DataFrame(
    [
        {
            "check": "four_branch_shapes_exact",
            "passed": all(
                x.shape
                == expected_shape_628hc3a3a2r4b
                for x
                in branch_matrices_628hc3a3a2r4b.values()
            ),
        },
        {
            "check": "four_branch_values_finite",
            "passed": all(
                np.isfinite(x).all()
                for x
                in branch_matrices_628hc3a3a2r4b.values()
            ),
        },
        {
            "check": "four_branch_values_within_0_1",
            "passed": all(
                np.min(x) >= 0.0
                and np.max(x) <= 1.0
                for x
                in branch_matrices_628hc3a3a2r4b.values()
            ),
        },
        {
            "check": "pre_global_shape_exact",
            "passed": (
                pre_global_matrix_628hc3a3a2r4b.shape
                == expected_shape_628hc3a3a2r4b
            ),
        },
        {
            "check": "scalar_shape_exact",
            "passed": (
                reconstructed_scalar_628hc3a3a2r4b.shape
                == expected_shape_628hc3a3a2r4b
            ),
        },
        {
            "check": "frozen_C2C_scalar_risk_reproduced",
            "passed": exact_1e12_628hc3a3a2r4b,
        },
    ]
)

print("\n--- FINAL R4B AUDIT ---")
display(audit_628hc3a3a2r4b)

all_checks_passed_628hc3a3a2r4b = bool(
    audit_628hc3a3a2r4b[
        "passed"
    ].all()
)


# ------------------------------------------------
# 7. SAFETY CONTRACT
# ------------------------------------------------

print("\n--- SAFETY CONTRACT ---")
print("DEV RUL labels used       : False")
print("Official TEST labels read : False")
print("TEST scored               : False")
print("Threshold selected        : False")
print("Adaptive fusion selected  : False")
print("Model executed            : False")
print("Model retrained           : False")
print("Files written             : False")


# ------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------

print(
    "\n" + "=" * 104
)
print(
    "STEP 6.28H-C3-A3-A2-F0-R4B FINAL SUMMARY"
)
print(
    "=" * 104
)

print(
    "Four calibrated branches shape :",
    expected_shape_628hc3a3a2r4b,
)

print(
    "Pre-global matrix shape         :",
    pre_global_matrix_628hc3a3a2r4b.shape,
)

print(
    "Reconstructed scalar shape      :",
    reconstructed_scalar_628hc3a3a2r4b.shape,
)

print(
    "Maximum reconstruction error    :",
    max_abs_error_628hc3a3a2r4b,
)

print(
    "Mean reconstruction error       :",
    mean_abs_error_628hc3a3a2r4b,
)

print(
    "Frozen C2-C scalar reproduced   :",
    exact_1e12_628hc3a3a2r4b,
)

print(
    "All R4B checks passed           :",
    all_checks_passed_628hc3a3a2r4b,
)

print(
    "\n=== END STEP "
    "6.28H-C3-A3-A2-F0-R4B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F1-A
# FREEZE FOUR CALIBRATED BRANCH MATRICES TO DISK
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A2-F1-A — FREEZE FOUR CALIBRATED BRANCH MATRICES")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Recover the exact in-memory objects already validated
# ---------------------------------------------------------------

level_628hc3a3f1a = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

noise_628hc3a3f1a = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

dropout_628hc3a3f1a = np.asarray(
    FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3f1a = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

scalar_628hc3a3f1a = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

manifest_628hc3a3f1a = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

expected_shape_628hc3a3f1a = (315, 8367)

# ---------------------------------------------------------------
# 2. Hard safety checks before writing anything
# ---------------------------------------------------------------

for arr_628hc3a3f1a in (
    level_628hc3a3f1a,
    noise_628hc3a3f1a,
    dropout_628hc3a3f1a,
    stuck_628hc3a3f1a,
    scalar_628hc3a3f1a,
):
    if arr_628hc3a3f1a.shape != expected_shape_628hc3a3f1a:
        raise RuntimeError(
            f"Unexpected matrix shape: {arr_628hc3a3f1a.shape}"
        )

    if not np.isfinite(arr_628hc3a3f1a).all():
        raise RuntimeError(
            "A matrix contains non-finite values."
        )

    if not np.all(
        (arr_628hc3a3f1a >= 0.0)
        & (arr_628hc3a3f1a <= 1.0)
    ):
        raise RuntimeError(
            "A matrix contains values outside [0,1]."
        )

if len(manifest_628hc3a3f1a) != 315:
    raise RuntimeError(
        "FD002 corruption manifest does not contain 315 rows."
    )

if manifest_628hc3a3f1a["scenario_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate scenario_id values detected."
    )

scenario_numbers_628hc3a3f1a = (
    manifest_628hc3a3f1a["scenario_number"]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    scenario_numbers_628hc3a3f1a,
    np.arange(1, 316, dtype=np.int64),
):
    raise RuntimeError(
        "Manifest row order is not scenario_number 1..315. "
        "Do not write branch artifacts."
    )

scenario_ids_628hc3a3f1a = (
    manifest_628hc3a3f1a["scenario_id"]
    .astype(str)
    .to_numpy(dtype="U")
)

# ---------------------------------------------------------------
# 3. Locate the existing project structure
# ---------------------------------------------------------------

cwd_628hc3a3f1a = Path.cwd()

if (cwd_628hc3a3f1a / "06_results").exists():
    project_root_628hc3a3f1a = cwd_628hc3a3f1a

elif (cwd_628hc3a3f1a.parent / "06_results").exists():
    project_root_628hc3a3f1a = cwd_628hc3a3f1a.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc3a3f1a = (
    project_root_628hc3a3f1a
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

freeze_dir_628hc3a3f1a.mkdir(
    parents=True,
    exist_ok=True,
)

npz_path_628hc3a3f1a = (
    freeze_dir_628hc3a3f1a
    / "fd002_c3a3_four_calibrated_branch_risks.npz"
)

manifest_path_628hc3a3f1a = (
    freeze_dir_628hc3a3f1a
    / "fd002_c3a3_corruption_manifest.csv"
)

metadata_path_628hc3a3f1a = (
    freeze_dir_628hc3a3f1a
    / "fd002_c3a3_four_calibrated_branch_risks_metadata.json"
)

# Do not silently overwrite a previous freeze.
existing_paths_628hc3a3f1a = [
    path_628hc3a3f1a
    for path_628hc3a3f1a in (
        npz_path_628hc3a3f1a,
        manifest_path_628hc3a3f1a,
        metadata_path_628hc3a3f1a,
    )
    if path_628hc3a3f1a.exists()
]

if existing_paths_628hc3a3f1a:
    raise FileExistsError(
        "Freeze artifact already exists. "
        "Nothing was overwritten:\n"
        + "\n".join(
            str(path_628hc3a3f1a)
            for path_628hc3a3f1a
            in existing_paths_628hc3a3f1a
        )
    )

# ---------------------------------------------------------------
# 4. Stable SHA-256 helper for arrays
# ---------------------------------------------------------------

def sha256_array_628hc3a3f1a(array):
    array = np.ascontiguousarray(
        np.asarray(array)
    )

    return hashlib.sha256(
        array.tobytes(order="C")
    ).hexdigest()


array_hashes_628hc3a3f1a = {
    "level":
        sha256_array_628hc3a3f1a(
            level_628hc3a3f1a
        ),

    "noise":
        sha256_array_628hc3a3f1a(
            noise_628hc3a3f1a
        ),

    "dropout":
        sha256_array_628hc3a3f1a(
            dropout_628hc3a3f1a
        ),

    "stuck":
        sha256_array_628hc3a3f1a(
            stuck_628hc3a3f1a
        ),

    "sensor_quality_risk":
        sha256_array_628hc3a3f1a(
            scalar_628hc3a3f1a
        ),
}

# ---------------------------------------------------------------
# 5. Write the frozen numerical artifact
# ---------------------------------------------------------------

np.savez_compressed(
    npz_path_628hc3a3f1a,

    level=level_628hc3a3f1a,
    noise=noise_628hc3a3f1a,
    dropout=dropout_628hc3a3f1a,
    stuck=stuck_628hc3a3f1a,

    sensor_quality_risk=
        scalar_628hc3a3f1a,

    scenario_number=
        scenario_numbers_628hc3a3f1a,

    scenario_id=
        scenario_ids_628hc3a3f1a,
)

# ---------------------------------------------------------------
# 6. Write the exact row-order manifest used by the matrices
# ---------------------------------------------------------------

manifest_628hc3a3f1a.to_csv(
    manifest_path_628hc3a3f1a,
    index=False,
)

# ---------------------------------------------------------------
# 7. File-level SHA-256 helper
# ---------------------------------------------------------------

def sha256_file_628hc3a3f1a(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


npz_file_hash_628hc3a3f1a = (
    sha256_file_628hc3a3f1a(
        npz_path_628hc3a3f1a
    )
)

manifest_file_hash_628hc3a3f1a = (
    sha256_file_628hc3a3f1a(
        manifest_path_628hc3a3f1a
    )
)

# ---------------------------------------------------------------
# 8. Write lineage / reproducibility metadata
# ---------------------------------------------------------------

metadata_628hc3a3f1a = {
    "dataset":
        "NASA C-MAPSS FD002",

    "step":
        "6.28H-C3-A3-A2-F1-A",

    "artifact_status":
        "FROZEN_WRITE_COMPLETE_RELOAD_NOT_YET_VERIFIED",

    "branch_stage":
        "FIT-calibrated branch risks",

    "branch_names": [
        "level",
        "noise",
        "dropout",
        "stuck",
    ],

    "matrix_shape":
        list(
            expected_shape_628hc3a3f1a
        ),

    "matrix_dtype":
        "float64",

    "scenario_count":
        315,

    "dev_windows_per_scenario":
        8367,

    "source_objects": {
        "level":
            "FD002_LEVEL_RISK_MATRIX_628HC3A3A2",

        "noise":
            "FD002_NOISE_RISK_MATRIX_628HC3A3A2",

        "dropout":
            "FD002_DROPOUT_RISK_MATRIX_628HC3A3A2",

        "stuck":
            "FD002_STUCK_RISK_MATRIX_628HC3A3A2",

        "sensor_quality_risk":
            "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",

        "manifest":
            "FD002_CORRUPTION_MANIFEST_628GD",
    },

    "lineage_contract": {
        "c3a3a2_original_scalar_reconstruction_max_error":
            0.0,

        "r4a_confirmed_final_function_expects":
            "raw branch scores",

        "stored_branch_matrices_are":
            "already calibrated",

        "must_not_reapply_final_sensor_quality_from_raw_628hc2b":
            True,
    },

    "array_sha256":
        array_hashes_628hc3a3f1a,

    "file_sha256": {
        "npz":
            npz_file_hash_628hc3a3f1a,

        "manifest_csv":
            manifest_file_hash_628hc3a3f1a,
    },

    "methodological_status": {
        "dev_rul_labels_used_for_this_freeze":
            False,

        "official_test_labels_read":
            False,

        "test_scored":
            False,

        "adaptive_fusion_selected":
            False,

        "abstention_threshold_selected":
            False,

        "model_executed":
            False,

        "model_retrained":
            False,
    },
}

with open(
    metadata_path_628hc3a3f1a,
    "w",
    encoding="utf-8",
) as metadata_file_628hc3a3f1a:

    json.dump(
        metadata_628hc3a3f1a,
        metadata_file_628hc3a3f1a,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------
# 9. Write-only closure checks
# ---------------------------------------------------------------

written_paths_628hc3a3f1a = [
    npz_path_628hc3a3f1a,
    manifest_path_628hc3a3f1a,
    metadata_path_628hc3a3f1a,
]

all_files_exist_628hc3a3f1a = all(
    path.exists()
    and path.stat().st_size > 0
    for path in written_paths_628hc3a3f1a
)

if not all_files_exist_628hc3a3f1a:
    raise RuntimeError(
        "One or more frozen files were not written correctly."
    )

# ---------------------------------------------------------------
# 10. Final summary
# ---------------------------------------------------------------

print("\n--- FREEZE DIRECTORY ---")
print(freeze_dir_628hc3a3f1a)

print("\n--- WRITTEN ARTIFACTS ---")

for path_628hc3a3f1a in written_paths_628hc3a3f1a:
    print(
        path_628hc3a3f1a.name,
        "| exists =",
        path_628hc3a3f1a.exists(),
        "| bytes =",
        path_628hc3a3f1a.stat().st_size,
    )

print("\n--- ARRAY SHA-256 ---")

for (
    name_628hc3a3f1a,
    hash_628hc3a3f1a,
) in array_hashes_628hc3a3f1a.items():

    print(
        f"{name_628hc3a3f1a:<20}",
        hash_628hc3a3f1a,
    )

print("\n--- F1-A STATUS ---")
print("All files written correctly :", all_files_exist_628hc3a3f1a)
print("Independent reload verified : False")
print("DEV RUL labels used          : False")
print("Official TEST labels read    : False")
print("TEST scored                  : False")
print("Adaptive fusion selected     : False")
print("Threshold selected           : False")
print("Model executed               : False")
print("Model retrained              : False")
print("Files written                : True")

print(
    "\nNext required step: "
    "F1-B independent reload and exact verification."
)

print(
    "\n=== END STEP 6.28H-C3-A3-A2-F1-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A2-F1-B
# INDEPENDENT RELOAD + EXACT FROZEN-ARTIFACT VERIFICATION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A2-F1-B — INDEPENDENT RELOAD + EXACT VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Resolve the frozen artifact paths independently
# ---------------------------------------------------------------

cwd_628hc3a3f1b = Path.cwd()

if (cwd_628hc3a3f1b / "06_results").exists():
    project_root_628hc3a3f1b = cwd_628hc3a3f1b

elif (cwd_628hc3a3f1b.parent / "06_results").exists():
    project_root_628hc3a3f1b = cwd_628hc3a3f1b.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc3a3f1b = (
    project_root_628hc3a3f1b
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

npz_path_628hc3a3f1b = (
    freeze_dir_628hc3a3f1b
    / "fd002_c3a3_four_calibrated_branch_risks.npz"
)

manifest_path_628hc3a3f1b = (
    freeze_dir_628hc3a3f1b
    / "fd002_c3a3_corruption_manifest.csv"
)

metadata_path_628hc3a3f1b = (
    freeze_dir_628hc3a3f1b
    / "fd002_c3a3_four_calibrated_branch_risks_metadata.json"
)

for path_628hc3a3f1b in (
    npz_path_628hc3a3f1b,
    manifest_path_628hc3a3f1b,
    metadata_path_628hc3a3f1b,
):
    if not path_628hc3a3f1b.exists():
        raise FileNotFoundError(
            f"Required frozen artifact is missing: {path_628hc3a3f1b}"
        )

# ---------------------------------------------------------------
# 2. Load metadata and frozen files
# ---------------------------------------------------------------

with open(
    metadata_path_628hc3a3f1b,
    "r",
    encoding="utf-8",
) as metadata_file_628hc3a3f1b:
    metadata_628hc3a3f1b = json.load(
        metadata_file_628hc3a3f1b
    )

loaded_npz_628hc3a3f1b = np.load(
    npz_path_628hc3a3f1b,
    allow_pickle=False,
)

loaded_manifest_628hc3a3f1b = pd.read_csv(
    manifest_path_628hc3a3f1b
)

# ---------------------------------------------------------------
# 3. Verify expected NPZ keys
# ---------------------------------------------------------------

expected_keys_628hc3a3f1b = {
    "level",
    "noise",
    "dropout",
    "stuck",
    "sensor_quality_risk",
    "scenario_number",
    "scenario_id",
}

actual_keys_628hc3a3f1b = set(
    loaded_npz_628hc3a3f1b.files
)

keys_exact_628hc3a3f1b = bool(
    actual_keys_628hc3a3f1b
    == expected_keys_628hc3a3f1b
)

if not keys_exact_628hc3a3f1b:
    raise RuntimeError(
        "Frozen NPZ key set does not match the expected contract."
    )

# ---------------------------------------------------------------
# 4. Recover reloaded arrays
# ---------------------------------------------------------------

level_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["level"],
    dtype=np.float64,
)

noise_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["noise"],
    dtype=np.float64,
)

dropout_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["dropout"],
    dtype=np.float64,
)

stuck_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["stuck"],
    dtype=np.float64,
)

scalar_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["sensor_quality_risk"],
    dtype=np.float64,
)

scenario_number_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["scenario_number"],
    dtype=np.int64,
)

scenario_id_reload_628hc3a3f1b = np.asarray(
    loaded_npz_628hc3a3f1b["scenario_id"]
).astype(str)

loaded_npz_628hc3a3f1b.close()

expected_shape_628hc3a3f1b = (315, 8367)

# ---------------------------------------------------------------
# 5. Structural and numerical contract
# ---------------------------------------------------------------

reloaded_arrays_628hc3a3f1b = {
    "level": level_reload_628hc3a3f1b,
    "noise": noise_reload_628hc3a3f1b,
    "dropout": dropout_reload_628hc3a3f1b,
    "stuck": stuck_reload_628hc3a3f1b,
    "sensor_quality_risk": scalar_reload_628hc3a3f1b,
}

shape_exact_628hc3a3f1b = all(
    arr_628hc3a3f1b.shape == expected_shape_628hc3a3f1b
    for arr_628hc3a3f1b
    in reloaded_arrays_628hc3a3f1b.values()
)

finite_exact_628hc3a3f1b = all(
    np.isfinite(arr_628hc3a3f1b).all()
    for arr_628hc3a3f1b
    in reloaded_arrays_628hc3a3f1b.values()
)

range_exact_628hc3a3f1b = all(
    np.all(
        (arr_628hc3a3f1b >= 0.0)
        & (arr_628hc3a3f1b <= 1.0)
    )
    for arr_628hc3a3f1b
    in reloaded_arrays_628hc3a3f1b.values()
)

# ---------------------------------------------------------------
# 6. Hash helpers
# ---------------------------------------------------------------

def sha256_array_628hc3a3f1b(array):
    array = np.ascontiguousarray(
        np.asarray(array)
    )

    return hashlib.sha256(
        array.tobytes(order="C")
    ).hexdigest()


def sha256_file_628hc3a3f1b(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file_handle:
        while True:
            block = file_handle.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ---------------------------------------------------------------
# 7. Verify file hashes stored in metadata
# ---------------------------------------------------------------

npz_file_hash_exact_628hc3a3f1b = bool(
    sha256_file_628hc3a3f1b(
        npz_path_628hc3a3f1b
    )
    ==
    metadata_628hc3a3f1b[
        "file_sha256"
    ][
        "npz"
    ]
)

manifest_file_hash_exact_628hc3a3f1b = bool(
    sha256_file_628hc3a3f1b(
        manifest_path_628hc3a3f1b
    )
    ==
    metadata_628hc3a3f1b[
        "file_sha256"
    ][
        "manifest_csv"
    ]
)

# ---------------------------------------------------------------
# 8. Verify every reloaded array hash
# ---------------------------------------------------------------

array_hash_checks_628hc3a3f1b = {
    name_628hc3a3f1b: bool(
        sha256_array_628hc3a3f1b(
            arr_628hc3a3f1b
        )
        ==
        metadata_628hc3a3f1b[
            "array_sha256"
        ][
            name_628hc3a3f1b
        ]
    )
    for (
        name_628hc3a3f1b,
        arr_628hc3a3f1b,
    )
    in reloaded_arrays_628hc3a3f1b.items()
}

all_array_hashes_exact_628hc3a3f1b = bool(
    all(
        array_hash_checks_628hc3a3f1b.values()
    )
)

# ---------------------------------------------------------------
# 9. Verify exact equality with the validated in-memory sources
# ---------------------------------------------------------------

memory_exact_628hc3a3f1b = {
    "level": bool(
        np.array_equal(
            level_reload_628hc3a3f1b,
            np.asarray(
                FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
                dtype=np.float64,
            ),
        )
    ),

    "noise": bool(
        np.array_equal(
            noise_reload_628hc3a3f1b,
            np.asarray(
                FD002_NOISE_RISK_MATRIX_628HC3A3A2,
                dtype=np.float64,
            ),
        )
    ),

    "dropout": bool(
        np.array_equal(
            dropout_reload_628hc3a3f1b,
            np.asarray(
                FD002_DROPOUT_RISK_MATRIX_628HC3A3A2,
                dtype=np.float64,
            ),
        )
    ),

    "stuck": bool(
        np.array_equal(
            stuck_reload_628hc3a3f1b,
            np.asarray(
                FD002_STUCK_RISK_MATRIX_628HC3A3A2,
                dtype=np.float64,
            ),
        )
    ),

    "sensor_quality_risk": bool(
        np.array_equal(
            scalar_reload_628hc3a3f1b,
            np.asarray(
                FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
                dtype=np.float64,
            ),
        )
    ),
}

all_memory_exact_628hc3a3f1b = bool(
    all(
        memory_exact_628hc3a3f1b.values()
    )
)

# ---------------------------------------------------------------
# 10. Verify scenario ordering against CSV and memory manifest
# ---------------------------------------------------------------

csv_scenario_number_628hc3a3f1b = (
    loaded_manifest_628hc3a3f1b[
        "scenario_number"
    ]
    .to_numpy(dtype=np.int64)
)

csv_scenario_id_628hc3a3f1b = (
    loaded_manifest_628hc3a3f1b[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)

memory_manifest_628hc3a3f1b = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .reset_index(drop=True)
)

memory_scenario_number_628hc3a3f1b = (
    memory_manifest_628hc3a3f1b[
        "scenario_number"
    ]
    .to_numpy(dtype=np.int64)
)

memory_scenario_id_628hc3a3f1b = (
    memory_manifest_628hc3a3f1b[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)

scenario_number_exact_628hc3a3f1b = bool(
    np.array_equal(
        scenario_number_reload_628hc3a3f1b,
        csv_scenario_number_628hc3a3f1b,
    )
    and
    np.array_equal(
        scenario_number_reload_628hc3a3f1b,
        memory_scenario_number_628hc3a3f1b,
    )
)

scenario_id_exact_628hc3a3f1b = bool(
    np.array_equal(
        scenario_id_reload_628hc3a3f1b,
        csv_scenario_id_628hc3a3f1b,
    )
    and
    np.array_equal(
        scenario_id_reload_628hc3a3f1b,
        memory_scenario_id_628hc3a3f1b,
    )
)

# ---------------------------------------------------------------
# 11. Verify metadata lineage contract
# ---------------------------------------------------------------

metadata_contract_exact_628hc3a3f1b = bool(
    metadata_628hc3a3f1b["dataset"]
    == "NASA C-MAPSS FD002"

    and metadata_628hc3a3f1b[
        "branch_stage"
    ]
    == "FIT-calibrated branch risks"

    and metadata_628hc3a3f1b[
        "matrix_shape"
    ]
    == [315, 8367]

    and metadata_628hc3a3f1b[
        "lineage_contract"
    ][
        "c3a3a2_original_scalar_reconstruction_max_error"
    ]
    == 0.0

    and metadata_628hc3a3f1b[
        "lineage_contract"
    ][
        "must_not_reapply_final_sensor_quality_from_raw_628hc2b"
    ]
    is True
)

# ---------------------------------------------------------------
# 12. Final audit
# ---------------------------------------------------------------

audit_rows_628hc3a3f1b = [
    ("npz_keys_exact", keys_exact_628hc3a3f1b),
    ("all_matrix_shapes_exact", shape_exact_628hc3a3f1b),
    ("all_values_finite", finite_exact_628hc3a3f1b),
    ("all_values_within_0_1", range_exact_628hc3a3f1b),
    ("npz_file_hash_exact", npz_file_hash_exact_628hc3a3f1b),
    ("manifest_file_hash_exact", manifest_file_hash_exact_628hc3a3f1b),
    ("all_array_hashes_exact", all_array_hashes_exact_628hc3a3f1b),
    ("all_arrays_exact_vs_memory", all_memory_exact_628hc3a3f1b),
    ("scenario_number_order_exact", scenario_number_exact_628hc3a3f1b),
    ("scenario_id_order_exact", scenario_id_exact_628hc3a3f1b),
    ("metadata_lineage_contract_exact", metadata_contract_exact_628hc3a3f1b),
]

audit_628hc3a3f1b = pd.DataFrame(
    audit_rows_628hc3a3f1b,
    columns=[
        "check",
        "passed",
    ],
)

print("\n--- F1-B INDEPENDENT RELOAD AUDIT ---")
display(audit_628hc3a3f1b)

print("\n--- ARRAY HASH VERIFICATION ---")

for (
    name_628hc3a3f1b,
    passed_628hc3a3f1b,
) in array_hash_checks_628hc3a3f1b.items():

    print(
        f"{name_628hc3a3f1b:<20}",
        passed_628hc3a3f1b,
    )

print("\n--- EXACT RELOAD VS MEMORY ---")

for (
    name_628hc3a3f1b,
    passed_628hc3a3f1b,
) in memory_exact_628hc3a3f1b.items():

    print(
        f"{name_628hc3a3f1b:<20}",
        passed_628hc3a3f1b,
    )

all_checks_passed_628hc3a3f1b = bool(
    audit_628hc3a3f1b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3f1b:
    raise RuntimeError(
        "Independent frozen-artifact verification failed."
    )

# ---------------------------------------------------------------
# 13. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C3-A3-A2-F1-B FINAL SUMMARY")
print("=" * 94)

print(
    "Frozen branch matrix shape       :",
    expected_shape_628hc3a3f1b,
)

print(
    "All array hashes exact           :",
    all_array_hashes_exact_628hc3a3f1b,
)

print(
    "All arrays exact vs memory       :",
    all_memory_exact_628hc3a3f1b,
)

print(
    "Scenario order exact             :",
    bool(
        scenario_number_exact_628hc3a3f1b
        and scenario_id_exact_628hc3a3f1b
    ),
)

print(
    "File hashes exact                :",
    bool(
        npz_file_hash_exact_628hc3a3f1b
        and manifest_file_hash_exact_628hc3a3f1b
    ),
)

print(
    "Metadata lineage contract exact  :",
    metadata_contract_exact_628hc3a3f1b,
)

print(
    "Independent reload verified      :",
    all_checks_passed_628hc3a3f1b,
)

print("DEV RUL labels used              : False")
print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Adaptive fusion selected         : False")
print("Threshold selected               : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files modified                   : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A2-F1-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A0
# LABEL-FREE BRANCH-PATTERN ACTIVATION AUDIT
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A0 — LABEL-FREE BRANCH-PATTERN ACTIVATION AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Load the independently verified frozen branch artifact
# ---------------------------------------------------------------

cwd_628hc3a3a3a0 = Path.cwd()

if (cwd_628hc3a3a3a0 / "06_results").exists():
    project_root_628hc3a3a3a0 = cwd_628hc3a3a3a0

elif (cwd_628hc3a3a3a0.parent / "06_results").exists():
    project_root_628hc3a3a3a0 = cwd_628hc3a3a3a0.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc3a3a3a0 = (
    project_root_628hc3a3a3a0
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

npz_path_628hc3a3a3a0 = (
    freeze_dir_628hc3a3a3a0
    / "fd002_c3a3_four_calibrated_branch_risks.npz"
)

manifest_path_628hc3a3a3a0 = (
    freeze_dir_628hc3a3a3a0
    / "fd002_c3a3_corruption_manifest.csv"
)

if not npz_path_628hc3a3a3a0.exists():
    raise FileNotFoundError(
        f"Frozen branch artifact missing: "
        f"{npz_path_628hc3a3a3a0}"
    )

if not manifest_path_628hc3a3a3a0.exists():
    raise FileNotFoundError(
        f"Frozen manifest missing: "
        f"{manifest_path_628hc3a3a3a0}"
    )

with np.load(
    npz_path_628hc3a3a3a0,
    allow_pickle=False,
) as frozen_628hc3a3a3a0:

    level_628hc3a3a3a0 = np.asarray(
        frozen_628hc3a3a3a0["level"],
        dtype=np.float64,
    )

    noise_628hc3a3a3a0 = np.asarray(
        frozen_628hc3a3a3a0["noise"],
        dtype=np.float64,
    )

    dropout_628hc3a3a3a0 = np.asarray(
        frozen_628hc3a3a3a0["dropout"],
        dtype=np.float64,
    )

    stuck_628hc3a3a3a0 = np.asarray(
        frozen_628hc3a3a3a0["stuck"],
        dtype=np.float64,
    )

manifest_628hc3a3a3a0 = (
    pd.read_csv(
        manifest_path_628hc3a3a3a0
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_shape_628hc3a3a3a0 = (
    315,
    8367,
)

for matrix_628hc3a3a3a0 in (
    level_628hc3a3a3a0,
    noise_628hc3a3a3a0,
    dropout_628hc3a3a3a0,
    stuck_628hc3a3a3a0,
):

    if (
        matrix_628hc3a3a3a0.shape
        !=
        expected_shape_628hc3a3a3a0
    ):
        raise RuntimeError(
            "Unexpected frozen branch shape: "
            f"{matrix_628hc3a3a3a0.shape}"
        )

if not np.array_equal(
    manifest_628hc3a3a3a0[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen manifest scenario order is not 1..315."
    )

# ---------------------------------------------------------------
# 2. Recover clean calibrated DEV branches
# ---------------------------------------------------------------

required_clean_keys_628hc3a3a3a0 = {
    "level",
    "noise",
    "dropout",
    "stuck",
}

if not (
    required_clean_keys_628hc3a3a3a0
    .issubset(
        CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B.keys()
    )
):
    raise RuntimeError(
        "Clean calibrated branch dictionary is incomplete."
    )

clean_level_628hc3a3a3a0 = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "level"
    ],
    dtype=np.float64,
).reshape(-1)

clean_noise_628hc3a3a3a0 = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "noise"
    ],
    dtype=np.float64,
).reshape(-1)

clean_dropout_628hc3a3a3a0 = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "dropout"
    ],
    dtype=np.float64,
).reshape(-1)

clean_stuck_628hc3a3a3a0 = np.asarray(
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B[
        "stuck"
    ],
    dtype=np.float64,
).reshape(-1)

if not all(
    arr_628hc3a3a3a0.shape
    ==
    (8367,)
    for arr_628hc3a3a3a0
    in (
        clean_level_628hc3a3a3a0,
        clean_noise_628hc3a3a3a0,
        clean_dropout_628hc3a3a3a0,
        clean_stuck_628hc3a3a3a0,
    )
):
    raise RuntimeError(
        "Unexpected clean calibrated branch shape."
    )

# ---------------------------------------------------------------
# 3. Diagnostic branch patterns
#
# IMPORTANT:
# These are NOT selected gates.
# They are label-free conjunction diagnostics only.
# ---------------------------------------------------------------

HIGH_BRANCH_MARKER_628HC3A3A3A0 = 0.95

corrupted_patterns_628hc3a3a3a0 = {

    "stuck_only":
        stuck_628hc3a3a3a0,

    "noise_and_stuck":
        np.minimum(
            noise_628hc3a3a3a0,
            stuck_628hc3a3a3a0,
        ),

    "level_and_stuck":
        np.minimum(
            level_628hc3a3a3a0,
            stuck_628hc3a3a3a0,
        ),

    "level_noise_stuck":
        np.minimum.reduce(
            [
                level_628hc3a3a3a0,
                noise_628hc3a3a3a0,
                stuck_628hc3a3a3a0,
            ]
        ),
}

clean_patterns_628hc3a3a3a0 = {

    "stuck_only":
        clean_stuck_628hc3a3a3a0,

    "noise_and_stuck":
        np.minimum(
            clean_noise_628hc3a3a3a0,
            clean_stuck_628hc3a3a3a0,
        ),

    "level_and_stuck":
        np.minimum(
            clean_level_628hc3a3a3a0,
            clean_stuck_628hc3a3a3a0,
        ),

    "level_noise_stuck":
        np.minimum.reduce(
            [
                clean_level_628hc3a3a3a0,
                clean_noise_628hc3a3a3a0,
                clean_stuck_628hc3a3a3a0,
            ]
        ),
}

# ---------------------------------------------------------------
# 4. Clean false-activation diagnostic
# ---------------------------------------------------------------

clean_rows_628hc3a3a3a0 = []

for (
    pattern_name_628hc3a3a3a0,
    pattern_values_628hc3a3a3a0,
) in clean_patterns_628hc3a3a3a0.items():

    clean_rows_628hc3a3a3a0.append(
        {
            "pattern":
                pattern_name_628hc3a3a3a0,

            "clean_median":
                float(
                    np.median(
                        pattern_values_628hc3a3a3a0
                    )
                ),

            "clean_q95":
                float(
                    np.quantile(
                        pattern_values_628hc3a3a3a0,
                        0.95,
                        method="linear",
                    )
                ),

            "clean_activation_fraction_ge_095":
                float(
                    np.mean(
                        pattern_values_628hc3a3a3a0
                        >=
                        HIGH_BRANCH_MARKER_628HC3A3A3A0
                    )
                ),
        }
    )

FD002_C3A3A3A0_CLEAN_PATTERN_AUDIT = (
    pd.DataFrame(
        clean_rows_628hc3a3a3a0
    )
)

print(
    "\n--- CLEAN DEV PATTERN ACTIVATION ---"
)

display(
    FD002_C3A3A3A0_CLEAN_PATTERN_AUDIT
)

# ---------------------------------------------------------------
# 5. Scenario-level branch-pattern response
# ---------------------------------------------------------------

scenario_pattern_df_628hc3a3a3a0 = (
    manifest_628hc3a3a3a0[
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .copy()
)

for (
    pattern_name_628hc3a3a3a0,
    pattern_matrix_628hc3a3a3a0,
) in corrupted_patterns_628hc3a3a3a0.items():

    scenario_pattern_df_628hc3a3a3a0[
        f"{pattern_name_628hc3a3a3a0}_median"
    ] = np.median(
        pattern_matrix_628hc3a3a3a0,
        axis=1,
    )

    scenario_pattern_df_628hc3a3a3a0[
        f"{pattern_name_628hc3a3a3a0}_activation_ge_095"
    ] = np.mean(
        pattern_matrix_628hc3a3a3a0
        >=
        HIGH_BRANCH_MARKER_628HC3A3A3A0,
        axis=1,
    )

FD002_C3A3A3A0_SCENARIO_PATTERN_RESPONSE = (
    scenario_pattern_df_628hc3a3a3a0
)

# ---------------------------------------------------------------
# 6. Family × severity diagnostic
# ---------------------------------------------------------------

pattern_metric_columns_628hc3a3a3a0 = [
    column_628hc3a3a3a0
    for column_628hc3a3a3a0
    in scenario_pattern_df_628hc3a3a3a0.columns
    if (
        column_628hc3a3a3a0.endswith(
            "_median"
        )
        or
        column_628hc3a3a3a0.endswith(
            "_activation_ge_095"
        )
    )
]

family_severity_628hc3a3a3a0 = (
    scenario_pattern_df_628hc3a3a3a0
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )[
        pattern_metric_columns_628hc3a3a3a0
    ]
    .median()
)

severity_order_628hc3a3a3a0 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_severity_628hc3a3a3a0[
    "_severity_order"
] = (
    family_severity_628hc3a3a3a0[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a0
    )
)

family_severity_628hc3a3a3a0 = (
    family_severity_628hc3a3a3a0
    .sort_values(
        [
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(
        drop=True
    )
)

FD002_C3A3A3A0_FAMILY_SEVERITY = (
    family_severity_628hc3a3a3a0
)

print(
    "\n--- FAMILY × SEVERITY: "
    "LABEL-FREE BRANCH-PATTERN ACTIVATION ---"
)

display(
    FD002_C3A3A3A0_FAMILY_SEVERITY
)

# ---------------------------------------------------------------
# 7. Strongest strict-pattern scenarios
# ---------------------------------------------------------------

strict_pattern_col_628hc3a3a3a0 = (
    "level_noise_stuck_activation_ge_095"
)

top_strict_628hc3a3a3a0 = (
    scenario_pattern_df_628hc3a3a3a0
    .sort_values(
        strict_pattern_col_628hc3a3a3a0,
        ascending=False,
        kind="mergesort",
    )
    .head(
        20
    )
    [
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            "level_noise_stuck_median",
            "level_noise_stuck_activation_ge_095",
        ]
    ]
    .reset_index(
        drop=True
    )
)

FD002_C3A3A3A0_TOP_STRICT_ACTIVATION = (
    top_strict_628hc3a3a3a0
)

print(
    "\n--- TOP 20 SCENARIOS: "
    "LEVEL + NOISE + STUCK CO-ACTIVATION ---"
)

display(
    FD002_C3A3A3A0_TOP_STRICT_ACTIVATION
)

# ---------------------------------------------------------------
# 8. Integrity audit
# ---------------------------------------------------------------

all_pattern_values_valid_628hc3a3a3a0 = bool(
    all(
        np.isfinite(
            matrix_628hc3a3a3a0
        ).all()
        and
        np.all(
            (
                matrix_628hc3a3a3a0
                >=
                0.0
            )
            &
            (
                matrix_628hc3a3a3a0
                <=
                1.0
            )
        )
        for matrix_628hc3a3a3a0
        in corrupted_patterns_628hc3a3a3a0.values()
    )
)

audit_628hc3a3a3a0 = pd.DataFrame(
    {
        "check": [
            "frozen_branch_shapes_exact",
            "manifest_scenario_count_315",
            "manifest_order_exact",
            "clean_branch_shapes_exact",
            "all_pattern_values_valid",
            "scenario_response_rows_315",
            "family_severity_groups_15",
        ],

        "passed": [

            all(
                matrix_628hc3a3a3a0.shape
                ==
                expected_shape_628hc3a3a3a0
                for matrix_628hc3a3a3a0
                in (
                    level_628hc3a3a3a0,
                    noise_628hc3a3a3a0,
                    dropout_628hc3a3a3a0,
                    stuck_628hc3a3a3a0,
                )
            ),

            len(
                manifest_628hc3a3a3a0
            )
            ==
            315,

            np.array_equal(
                manifest_628hc3a3a3a0[
                    "scenario_number"
                ].to_numpy(
                    dtype=np.int64
                ),
                np.arange(
                    1,
                    316,
                    dtype=np.int64,
                ),
            ),

            all(
                arr_628hc3a3a3a0.shape
                ==
                (8367,)
                for arr_628hc3a3a3a0
                in (
                    clean_level_628hc3a3a3a0,
                    clean_noise_628hc3a3a3a0,
                    clean_dropout_628hc3a3a3a0,
                    clean_stuck_628hc3a3a3a0,
                )
            ),

            all_pattern_values_valid_628hc3a3a3a0,

            len(
                FD002_C3A3A3A0_SCENARIO_PATTERN_RESPONSE
            )
            ==
            315,

            len(
                FD002_C3A3A3A0_FAMILY_SEVERITY
            )
            ==
            15,
        ],
    }
)

print(
    "\n--- C3-A3-A3-A0 AUDIT ---"
)

display(
    audit_628hc3a3a3a0
)

all_checks_passed_628hc3a3a3a0 = bool(
    audit_628hc3a3a3a0[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3a0:
    raise RuntimeError(
        "C3-A3-A3-A0 diagnostic audit failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A0 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Diagnostic high-branch marker :",
    HIGH_BRANCH_MARKER_628HC3A3A3A0,
)

print(
    "Diagnostic patterns evaluated :",
    len(
        corrupted_patterns_628hc3a3a3a0
    ),
)

print(
    "Scenarios evaluated           :",
    len(
        FD002_C3A3A3A0_SCENARIO_PATTERN_RESPONSE
    ),
)

print(
    "DEV windows/scenario          :",
    8367,
)

print(
    "All checks passed             :",
    all_checks_passed_628hc3a3a3a0,
)

print("DEV RUL labels used           : False")
print("Official TEST labels read     : False")
print("TEST scored                   : False")
print("Adaptive fusion rule selected : False")
print("Combination weight selected   : False")
print("Threshold selected            : False")
print("Model executed                : False")
print("Model retrained               : False")
print("Files written                 : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A0 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A1
# BRANCH-PATTERN UTILITY ALIGNMENT AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A1 — BRANCH-PATTERN UTILITY ALIGNMENT AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Required development-side diagnostic objects
# ---------------------------------------------------------------

pattern_df_628hc3a3a3a1 = (
    FD002_C3A3A3A0_SCENARIO_PATTERN_RESPONSE
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

fusion_compare_628hc3a3a3a1 = (
    FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

required_pattern_columns_628hc3a3a3a1 = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "stuck_only_activation_ge_095",
    "noise_and_stuck_activation_ge_095",
    "level_and_stuck_activation_ge_095",
    "level_noise_stuck_activation_ge_095",
    "level_noise_stuck_median",
}

missing_pattern_columns_628hc3a3a3a1 = (
    required_pattern_columns_628hc3a3a3a1
    -
    set(
        pattern_df_628hc3a3a3a1.columns
    )
)

if missing_pattern_columns_628hc3a3a3a1:
    raise RuntimeError(
        "Missing A0 pattern columns: "
        f"{sorted(missing_pattern_columns_628hc3a3a3a1)}"
    )

required_compare_columns_628hc3a3a3a1 = {
    "scenario_number",
    "delta_gain80_vs_uncertainty",
}

missing_compare_columns_628hc3a3a3a1 = (
    required_compare_columns_628hc3a3a3a1
    -
    set(
        fusion_compare_628hc3a3a3a1.columns
    )
)

if missing_compare_columns_628hc3a3a3a1:
    raise RuntimeError(
        "Missing C3-A2 comparison columns: "
        f"{sorted(missing_compare_columns_628hc3a3a3a1)}"
    )

if len(pattern_df_628hc3a3a3a1) != 315:
    raise RuntimeError(
        "Expected 315 branch-pattern rows."
    )

if len(fusion_compare_628hc3a3a3a1) != 315:
    raise RuntimeError(
        "Expected 315 C3-A2 comparison rows."
    )

if pattern_df_628hc3a3a3a1[
    "scenario_number"
].duplicated().any():
    raise RuntimeError(
        "Duplicate scenario numbers in branch-pattern table."
    )

if fusion_compare_628hc3a3a3a1[
    "scenario_number"
].duplicated().any():
    raise RuntimeError(
        "Duplicate scenario numbers in C3-A2 comparison table."
    )

# ---------------------------------------------------------------
# 2. Recover the diagnostic C3-A2 gamma
#
# This is NOT being selected or frozen here.
# ---------------------------------------------------------------

diagnostic_gamma_628hc3a3a3a1 = float(
    FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2
    .iloc[0][
        "gamma"
    ]
)

print(
    "\nC3-A2 diagnostic gamma used for utility comparison :",
    diagnostic_gamma_628hc3a3a3a1,
)

# ---------------------------------------------------------------
# 3. Merge branch activation with previously computed utility
# ---------------------------------------------------------------

utility_columns_628hc3a3a3a1 = (
    fusion_compare_628hc3a3a3a1[
        [
            "scenario_number",
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .copy()
)

alignment_df_628hc3a3a3a1 = (
    pattern_df_628hc3a3a3a1
    .merge(
        utility_columns_628hc3a3a3a1,
        on="scenario_number",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(alignment_df_628hc3a3a3a1) != 315:
    raise RuntimeError(
        "Merged alignment table does not contain 315 scenarios."
    )

if not np.isfinite(
    alignment_df_628hc3a3a3a1[
        "delta_gain80_vs_uncertainty"
    ].to_numpy(dtype=np.float64)
).all():
    raise RuntimeError(
        "Non-finite utility differences detected."
    )

FD002_C3A3A3A1_ALIGNMENT_628HC3A3A3A1 = (
    alignment_df_628hc3a3a3a1
)

# ---------------------------------------------------------------
# 4. Spearman association:
#    branch activation vs benefit of diagnostic fusion
# ---------------------------------------------------------------

pattern_columns_628hc3a3a3a1 = [
    "stuck_only_activation_ge_095",
    "noise_and_stuck_activation_ge_095",
    "level_and_stuck_activation_ge_095",
    "level_noise_stuck_activation_ge_095",
]

association_rows_628hc3a3a3a1 = []

for pattern_column_628hc3a3a3a1 in (
    pattern_columns_628hc3a3a3a1
):

    pair_628hc3a3a3a1 = (
        alignment_df_628hc3a3a3a1[
            [
                pattern_column_628hc3a3a3a1,
                "delta_gain80_vs_uncertainty",
            ]
        ]
    )

    spearman_628hc3a3a3a1 = float(
        pair_628hc3a3a3a1.corr(
            method="spearman"
        ).iloc[
            0,
            1,
        ]
    )

    association_rows_628hc3a3a3a1.append(
        {
            "pattern":
                pattern_column_628hc3a3a3a1,

            "spearman_with_delta_gain80":
                spearman_628hc3a3a3a1,
        }
    )

FD002_C3A3A3A1_PATTERN_ASSOCIATION_628HC3A3A3A1 = (
    pd.DataFrame(
        association_rows_628hc3a3a3a1
    )
)

print(
    "\n--- PATTERN ACTIVATION VS "
    "FUSION BENEFIT ASSOCIATION ---"
)

display(
    FD002_C3A3A3A1_PATTERN_ASSOCIATION_628HC3A3A3A1
)

# ---------------------------------------------------------------
# 5. Family × severity alignment
# ---------------------------------------------------------------

family_severity_rows_628hc3a3a3a1 = []

for (
    family_628hc3a3a3a1,
    severity_628hc3a3a3a1,
), group_628hc3a3a3a1 in (
    alignment_df_628hc3a3a3a1
    .groupby(
        [
            "family",
            "severity",
        ],
        observed=True,
    )
):

    delta_628hc3a3a3a1 = (
        group_628hc3a3a3a1[
            "delta_gain80_vs_uncertainty"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    strict_activation_628hc3a3a3a1 = (
        group_628hc3a3a3a1[
            "level_noise_stuck_activation_ge_095"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    family_severity_rows_628hc3a3a3a1.append(
        {
            "family":
                family_628hc3a3a3a1,

            "severity":
                severity_628hc3a3a3a1,

            "n_scenarios":
                len(
                    group_628hc3a3a3a1
                ),

            "median_strict_activation":
                float(
                    np.median(
                        strict_activation_628hc3a3a3a1
                    )
                ),

            "median_delta_gain80":
                float(
                    np.median(
                        delta_628hc3a3a3a1
                    )
                ),

            "mean_delta_gain80":
                float(
                    np.mean(
                        delta_628hc3a3a3a1
                    )
                ),

            "fraction_fusion_better":
                float(
                    np.mean(
                        delta_628hc3a3a3a1
                        >
                        0.0
                    )
                ),

            "fraction_fusion_worse":
                float(
                    np.mean(
                        delta_628hc3a3a3a1
                        <
                        0.0
                    )
                ),
        }
    )

family_severity_628hc3a3a3a1 = (
    pd.DataFrame(
        family_severity_rows_628hc3a3a3a1
    )
)

severity_order_628hc3a3a3a1 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_severity_628hc3a3a3a1[
    "_severity_order"
] = (
    family_severity_628hc3a3a3a1[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a1
    )
)

family_severity_628hc3a3a3a1 = (
    family_severity_628hc3a3a3a1
    .sort_values(
        [
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(
        drop=True
    )
)

FD002_C3A3A3A1_FAMILY_SEVERITY_628HC3A3A3A1 = (
    family_severity_628hc3a3a3a1
)

print(
    "\n--- FAMILY × SEVERITY: "
    "STRICT ACTIVATION VS FUSION UTILITY ---"
)

display(
    FD002_C3A3A3A1_FAMILY_SEVERITY_628HC3A3A3A1
)

# ---------------------------------------------------------------
# 6. Scenarios with strongest strict activation
# ---------------------------------------------------------------

top_alignment_628hc3a3a3a1 = (
    alignment_df_628hc3a3a3a1
    .sort_values(
        [
            "level_noise_stuck_activation_ge_095",
            "delta_gain80_vs_uncertainty",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .head(
        25
    )
    [
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            "level_noise_stuck_median",
            "level_noise_stuck_activation_ge_095",
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .reset_index(
        drop=True
    )
)

FD002_C3A3A3A1_TOP_ALIGNMENT_628HC3A3A3A1 = (
    top_alignment_628hc3a3a3a1
)

print(
    "\n--- TOP 25 STRICT-ACTIVATION SCENARIOS "
    "WITH UTILITY DELTA ---"
)

display(
    FD002_C3A3A3A1_TOP_ALIGNMENT_628HC3A3A3A1
)

# ---------------------------------------------------------------
# 7. Diagnostic split:
#    any strict high-branch co-activation vs none
#
# This is descriptive only.
# It is NOT a selected adaptive rule.
# ---------------------------------------------------------------

strict_activation_values_628hc3a3a3a1 = (
    alignment_df_628hc3a3a3a1[
        "level_noise_stuck_activation_ge_095"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

delta_values_628hc3a3a3a1 = (
    alignment_df_628hc3a3a3a1[
        "delta_gain80_vs_uncertainty"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

any_activation_mask_628hc3a3a3a1 = (
    strict_activation_values_628hc3a3a3a1
    >
    0.0
)

no_activation_mask_628hc3a3a3a1 = (
    ~any_activation_mask_628hc3a3a3a1
)

activation_split_rows_628hc3a3a3a1 = []

for (
    group_name_628hc3a3a3a1,
    mask_628hc3a3a3a1,
) in (
    (
        "no_strict_activation",
        no_activation_mask_628hc3a3a3a1,
    ),
    (
        "any_strict_activation",
        any_activation_mask_628hc3a3a3a1,
    ),
):

    group_delta_628hc3a3a3a1 = (
        delta_values_628hc3a3a3a1[
            mask_628hc3a3a3a1
        ]
    )

    activation_split_rows_628hc3a3a3a1.append(
        {
            "group":
                group_name_628hc3a3a3a1,

            "n_scenarios":
                int(
                    mask_628hc3a3a3a1.sum()
                ),

            "median_delta_gain80":
                float(
                    np.median(
                        group_delta_628hc3a3a3a1
                    )
                ),

            "mean_delta_gain80":
                float(
                    np.mean(
                        group_delta_628hc3a3a3a1
                    )
                ),

            "fraction_fusion_better":
                float(
                    np.mean(
                        group_delta_628hc3a3a3a1
                        >
                        0.0
                    )
                ),

            "fraction_fusion_worse":
                float(
                    np.mean(
                        group_delta_628hc3a3a3a1
                        <
                        0.0
                    )
                ),
        }
    )

FD002_C3A3A3A1_ACTIVATION_SPLIT_628HC3A3A3A1 = (
    pd.DataFrame(
        activation_split_rows_628hc3a3a3a1
    )
)

print(
    "\n--- DESCRIPTIVE STRICT-ACTIVATION SPLIT ---"
)

display(
    FD002_C3A3A3A1_ACTIVATION_SPLIT_628HC3A3A3A1
)

# ---------------------------------------------------------------
# 8. Integrity audit
# ---------------------------------------------------------------

audit_628hc3a3a3a1 = pd.DataFrame(
    {
        "check": [
            "pattern_rows_315",
            "fusion_comparison_rows_315",
            "merged_rows_315",
            "scenario_numbers_unique",
            "utility_delta_finite",
            "four_pattern_associations",
            "family_severity_groups_15",
        ],

        "passed": [
            len(
                pattern_df_628hc3a3a3a1
            )
            ==
            315,

            len(
                fusion_compare_628hc3a3a3a1
            )
            ==
            315,

            len(
                alignment_df_628hc3a3a3a1
            )
            ==
            315,

            bool(
                alignment_df_628hc3a3a3a1[
                    "scenario_number"
                ]
                .is_unique
            ),

            bool(
                np.isfinite(
                    delta_values_628hc3a3a3a1
                ).all()
            ),

            len(
                FD002_C3A3A3A1_PATTERN_ASSOCIATION_628HC3A3A3A1
            )
            ==
            4,

            len(
                FD002_C3A3A3A1_FAMILY_SEVERITY_628HC3A3A3A1
            )
            ==
            15,
        ],
    }
)

print(
    "\n--- C3-A3-A3-A1 AUDIT ---"
)

display(
    audit_628hc3a3a3a1
)

all_checks_passed_628hc3a3a3a1 = bool(
    audit_628hc3a3a3a1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3a1:
    raise RuntimeError(
        "C3-A3-A3-A1 utility-alignment audit failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

strict_spearman_628hc3a3a3a1 = float(
    FD002_C3A3A3A1_PATTERN_ASSOCIATION_628HC3A3A3A1
    .loc[
        FD002_C3A3A3A1_PATTERN_ASSOCIATION_628HC3A3A3A1[
            "pattern"
        ]
        ==
        "level_noise_stuck_activation_ge_095",
        "spearman_with_delta_gain80",
    ]
    .iloc[0]
)

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A1 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Diagnostic C3-A2 gamma              :",
    diagnostic_gamma_628hc3a3a3a1,
)

print(
    "Strict-pattern utility Spearman     :",
    strict_spearman_628hc3a3a3a1,
)

print(
    "Scenarios evaluated                 :",
    len(
        alignment_df_628hc3a3a3a1
    ),
)

print(
    "All checks passed                   :",
    all_checks_passed_628hc3a3a3a1,
)

print("DEV RUL labels used                 : True")
print("Official TEST labels read           : False")
print("TEST scored                         : False")
print("Adaptive fusion rule selected       : False")
print("Combination weight selected         : False")
print("Threshold selected                  : False")
print("Model executed                      : False")
print("Model retrained                     : False")
print("Files written                       : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A1 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A2
# ADAPTIVE-GATE ARCHITECTURE DIAGNOSTIC
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A2 — ADAPTIVE-GATE ARCHITECTURE DIAGNOSTIC")
print("=" * 94)

# 1. Frozen inputs
manifest = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

pred = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64,
)

u = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

q = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

level = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

noise = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

y_dev = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64,
).reshape(-1)

clean_pred = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
).reshape(-1)

clean_u = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_q_dict = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

clean_q = np.asarray(
    clean_q_dict["sensor_quality_risk"],
    dtype=np.float64,
).reshape(-1)

clean_level = np.asarray(
    clean_q_dict["level"],
    dtype=np.float64,
).reshape(-1)

clean_noise = np.asarray(
    clean_q_dict["noise"],
    dtype=np.float64,
).reshape(-1)

clean_stuck = np.asarray(
    clean_q_dict["stuck"],
    dtype=np.float64,
).reshape(-1)

expected_shape = (315, 8367)

for arr in (
    pred,
    u,
    q,
    level,
    noise,
    stuck,
):
    if arr.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected matrix shape: {arr.shape}"
        )

if len(manifest) != 315 or len(y_dev) != 8367:
    raise RuntimeError(
        "Population contract failed."
    )

if not np.array_equal(
    manifest["scenario_number"].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Scenario order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 2. Fixed architecture constants
# ---------------------------------------------------------------

GAMMA = 0.80
GATE_MARKER = 0.95
COVERAGE = 0.80

QUALITY_Q95 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)

ns = np.minimum(
    noise,
    stuck,
)

lns = np.minimum.reduce(
    [
        level,
        noise,
        stuck,
    ]
)

clean_ns = np.minimum(
    clean_noise,
    clean_stuck,
)

clean_lns = np.minimum.reduce(
    [
        clean_level,
        clean_noise,
        clean_stuck,
    ]
)

CANDIDATES = [
    (
        "uncertainty_only",
        "none",
        "fixed",
        0.00,
    ),
    (
        "fixed_global_g080",
        "none",
        "fixed",
        0.80,
    ),
    (
        "noise_stuck_hard095_g080",
        "ns",
        "hard",
        0.80,
    ),
    (
        "noise_stuck_soft095_g080",
        "ns",
        "soft",
        0.80,
    ),
    (
        "level_noise_stuck_hard095_g080",
        "lns",
        "hard",
        0.80,
    ),
    (
        "level_noise_stuck_soft095_g080",
        "lns",
        "soft",
        0.80,
    ),
]

print("\n--- CANDIDATE ARCHITECTURES ---")

display(
    pd.DataFrame(
        CANDIDATES,
        columns=[
            "candidate",
            "pattern",
            "gate_mode",
            "gamma",
        ],
    )
)

# ---------------------------------------------------------------
# 3. Helpers
# ---------------------------------------------------------------

def make_gate(
    score,
    mode,
):
    score = np.asarray(
        score,
        dtype=np.float64,
    )

    if mode == "fixed":
        return np.ones_like(
            score
        )

    if mode == "hard":
        return (
            score
            >=
            GATE_MARKER
        ).astype(
            np.float64
        )

    if mode == "soft":
        return np.clip(
            (
                score
                -
                GATE_MARKER
            )
            /
            (
                1.0
                -
                GATE_MARKER
            ),
            0.0,
            1.0,
        )

    raise ValueError(
        f"Unknown gate mode: {mode}"
    )


def resolve_gate(
    pattern,
    mode,
    i=None,
    clean=False,
):
    if pattern == "none":

        reference = (
            clean_u
            if clean
            else u[i]
        )

        return np.ones_like(
            reference
        )

    if pattern == "ns":

        score = (
            clean_ns
            if clean
            else ns[i]
        )

    elif pattern == "lns":

        score = (
            clean_lns
            if clean
            else lns[i]
        )

    else:
        raise ValueError(
            f"Unknown pattern: {pattern}"
        )

    return make_gate(
        score,
        mode,
    )


def adaptive_risk(
    u_values,
    q_values,
    gate_values,
    gamma,
):
    q_excess = (
        quality_excess_628hc3a2(
            q_values,
            QUALITY_Q95,
        )
    )

    return np.clip(
        u_values
        +
        (
            1.0
            -
            u_values
        )
        *
        float(
            gamma
        )
        *
        gate_values
        *
        q_excess,
        0.0,
        1.0,
    )


def evaluate_gain80(
    risk,
    absolute_error,
):
    result = (
        selective_mae_tie_aware_628hc3a2(
            risk,
            absolute_error,
            COVERAGE,
        )
    )

    full_mae = float(
        np.mean(
            absolute_error
        )
    )

    selective_mae = float(
        result[
            "mae"
        ]
    )

    gain = float(
        full_mae
        -
        selective_mae
    )

    return (
        full_mae,
        selective_mae,
        gain,
    )

# ---------------------------------------------------------------
# 4. Clean DEV response
# ---------------------------------------------------------------

clean_error = np.abs(
    clean_pred
    -
    y_dev
)

clean_rows = []

for (
    candidate,
    pattern,
    mode,
    gamma,
) in CANDIDATES:

    gate_values = resolve_gate(
        pattern,
        mode,
        clean=True,
    )

    risk = adaptive_risk(
        clean_u,
        clean_q,
        gate_values,
        gamma,
    )

    (
        full_mae,
        selective_mae,
        gain,
    ) = evaluate_gain80(
        risk,
        clean_error,
    )

    clean_rows.append(
        {
            "candidate":
                candidate,

            "clean_full_mae":
                full_mae,

            "clean_selective_mae80":
                selective_mae,

            "clean_gain80":
                gain,

            "gate_mean":
                float(
                    np.mean(
                        gate_values
                    )
                ),

            "gate_nonzero_fraction":
                float(
                    np.mean(
                        gate_values
                        >
                        0.0
                    )
                ),
        }
    )

FD002_C3A3A3A2_CLEAN_ARCHITECTURE = (
    pd.DataFrame(
        clean_rows
    )
)

print(
    "\n--- CLEAN DEV ARCHITECTURE DIAGNOSTIC ---"
)

display(
    FD002_C3A3A3A2_CLEAN_ARCHITECTURE
)

# ---------------------------------------------------------------
# 5. Full 315-scenario response
# ---------------------------------------------------------------

scenario_rows = []

for (
    candidate,
    pattern,
    mode,
    gamma,
) in CANDIDATES:

    for i in range(
        315
    ):

        gate_values = resolve_gate(
            pattern,
            mode,
            i=i,
            clean=False,
        )

        risk = adaptive_risk(
            u[i],
            q[i],
            gate_values,
            gamma,
        )

        absolute_error = np.abs(
            pred[i]
            -
            y_dev
        )

        (
            full_mae,
            selective_mae,
            gain,
        ) = evaluate_gain80(
            risk,
            absolute_error,
        )

        row = manifest.iloc[i]

        scenario_rows.append(
            {
                "candidate":
                    candidate,

                "scenario_id":
                    str(
                        row[
                            "scenario_id"
                        ]
                    ),

                "scenario_number":
                    int(
                        row[
                            "scenario_number"
                        ]
                    ),

                "family":
                    str(
                        row[
                            "family"
                        ]
                    ),

                "sensor":
                    str(
                        row[
                            "sensor"
                        ]
                    ),

                "severity":
                    str(
                        row[
                            "severity"
                        ]
                    ),

                "full_mae":
                    full_mae,

                "selective_mae80":
                    selective_mae,

                "gain80":
                    gain,

                "gate_mean":
                    float(
                        np.mean(
                            gate_values
                        )
                    ),

                "gate_nonzero_fraction":
                    float(
                        np.mean(
                            gate_values
                            >
                            0.0
                        )
                    ),
            }
        )

FD002_C3A3A3A2_SCENARIO_ARCHITECTURE = (
    pd.DataFrame(
        scenario_rows
    )
)

if (
    len(
        FD002_C3A3A3A2_SCENARIO_ARCHITECTURE
    )
    !=
    1890
):
    raise RuntimeError(
        "Expected 1890 scenario-candidate rows."
    )

# ---------------------------------------------------------------
# 6. Candidate summary
# ---------------------------------------------------------------

baseline = (
    FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
        FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ]
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

baseline_gain = (
    baseline[
        "gain80"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

clean_lookup = (
    FD002_C3A3A3A2_CLEAN_ARCHITECTURE
    .set_index(
        "candidate"
    )
)

summary_rows = []

for (
    candidate,
    _,
    _,
    _,
) in CANDIDATES:

    cdf = (
        FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
            FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
                "candidate"
            ]
            ==
            candidate
        ]
        .sort_values(
            "scenario_number",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    delta = (
        cdf[
            "gain80"
        ]
        .to_numpy(
            dtype=np.float64
        )
        -
        baseline_gain
    )

    family_medians = (
        cdf
        .groupby(
            "family",
            observed=True,
        )[
            "gain80"
        ]
        .median()
    )

    summary_rows.append(
        {
            "candidate":
                candidate,

            "clean_gain80":
                float(
                    clean_lookup.loc[
                        candidate,
                        "clean_gain80",
                    ]
                ),

            "median_gain80":
                float(
                    cdf[
                        "gain80"
                    ]
                    .median()
                ),

            "mean_gain80":
                float(
                    cdf[
                        "gain80"
                    ]
                    .mean()
                ),

            "q10_gain80":
                float(
                    cdf[
                        "gain80"
                    ]
                    .quantile(
                        0.10
                    )
                ),

            "worst_family_median_gain80":
                float(
                    family_medians.min()
                ),

            "median_delta_vs_uncertainty":
                float(
                    np.median(
                        delta
                    )
                ),

            "mean_delta_vs_uncertainty":
                float(
                    np.mean(
                        delta
                    )
                ),

            "improved_vs_uncertainty":
                int(
                    np.sum(
                        delta
                        >
                        1e-12
                    )
                ),

            "equal_vs_uncertainty":
                int(
                    np.sum(
                        np.abs(
                            delta
                        )
                        <=
                        1e-12
                    )
                ),

            "worse_vs_uncertainty":
                int(
                    np.sum(
                        delta
                        <
                        -1e-12
                    )
                ),
        }
    )

FD002_C3A3A3A2_ARCHITECTURE_SUMMARY = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "worst_family_median_gain80",
            "q10_gain80",
            "median_delta_vs_uncertainty",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- ADAPTIVE-GATE ARCHITECTURE SUMMARY ---"
)

display(
    FD002_C3A3A3A2_ARCHITECTURE_SUMMARY
)

# ---------------------------------------------------------------
# 7. Stuck-family response
# ---------------------------------------------------------------

FD002_C3A3A3A2_STUCK_RESPONSE = (
    FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
        FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
            "family"
        ]
        ==
        "stuck"
    ]
    .groupby(
        [
            "candidate",
            "severity",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        n_scenarios=(
            "scenario_number",
            "size",
        ),

        median_gain80=(
            "gain80",
            "median",
        ),

        mean_gain80=(
            "gain80",
            "mean",
        ),

        median_gate_mean=(
            "gate_mean",
            "median",
        ),
    )
)

severity_order = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

FD002_C3A3A3A2_STUCK_RESPONSE[
    "_order"
] = (
    FD002_C3A3A3A2_STUCK_RESPONSE[
        "severity"
    ]
    .map(
        severity_order
    )
)

FD002_C3A3A3A2_STUCK_RESPONSE = (
    FD002_C3A3A3A2_STUCK_RESPONSE
    .sort_values(
        [
            "candidate",
            "_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- STUCK-FAMILY ARCHITECTURE RESPONSE ---"
)

display(
    FD002_C3A3A3A2_STUCK_RESPONSE
)

# ---------------------------------------------------------------
# 8. Audit
# ---------------------------------------------------------------

audit = pd.DataFrame(
    {
        "check": [
            "candidate_count_6",
            "scenario_count_315",
            "dev_windows_8367",
            "scenario_candidate_rows_1890",
            "all_metrics_finite",
            "clean_rows_6",
            "stuck_rows_18",
        ],

        "passed": [
            len(CANDIDATES)
            ==
            6,

            len(manifest)
            ==
            315,

            len(y_dev)
            ==
            8367,

            len(
                FD002_C3A3A3A2_SCENARIO_ARCHITECTURE
            )
            ==
            1890,

            bool(
                np.isfinite(
                    FD002_C3A3A3A2_SCENARIO_ARCHITECTURE[
                        [
                            "full_mae",
                            "selective_mae80",
                            "gain80",
                            "gate_mean",
                        ]
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                )
                .all()
            ),

            len(
                FD002_C3A3A3A2_CLEAN_ARCHITECTURE
            )
            ==
            6,

            len(
                FD002_C3A3A3A2_STUCK_RESPONSE
            )
            ==
            18,
        ],
    }
)

print(
    "\n--- C3-A3-A3-A2 AUDIT ---"
)

display(
    audit
)

all_checks_passed = bool(
    audit[
        "passed"
    ]
    .all()
)

if not all_checks_passed:
    raise RuntimeError(
        "Adaptive-gate architecture diagnostic failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A2 FINAL SUMMARY"
)

print(
    "=" * 94
)

print("Candidates evaluated              : 6")
print("Scenario-candidate evaluations    : 1890")
print("Diagnostic gamma                  :", GAMMA)
print("Diagnostic branch marker          :", GATE_MARKER)
print("Coverage evaluated                :", COVERAGE)
print("All checks passed                 :", all_checks_passed)

print("DEV RUL labels used               : True")
print("Official TEST labels read         : False")
print("TEST scored                       : False")
print("Adaptive fusion rule selected     : False")
print("Combination weight selected       : False")
print("Abstention threshold selected     : False")
print("Model executed                    : False")
print("Model retrained                   : False")
print("Files written                     : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A2 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A3
# UNCERTAINTY–BRANCH DISAGREEMENT AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A3 — UNCERTAINTY–BRANCH DISAGREEMENT AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Frozen aligned inputs
# ---------------------------------------------------------------

manifest_628hc3a3a3a3 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

u_628hc3a3a3a3 = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

level_628hc3a3a3a3 = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

noise_628hc3a3a3a3 = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3a3a3 = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

clean_u_628hc3a3a3a3 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_branch_dict_628hc3a3a3a3 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

clean_level_628hc3a3a3a3 = np.asarray(
    clean_branch_dict_628hc3a3a3a3["level"],
    dtype=np.float64,
).reshape(-1)

clean_noise_628hc3a3a3a3 = np.asarray(
    clean_branch_dict_628hc3a3a3a3["noise"],
    dtype=np.float64,
).reshape(-1)

clean_stuck_628hc3a3a3a3 = np.asarray(
    clean_branch_dict_628hc3a3a3a3["stuck"],
    dtype=np.float64,
).reshape(-1)

utility_628hc3a3a3a3 = (
    FD002_C3A3A3A1_ALIGNMENT_628HC3A3A3A1[
        [
            "scenario_number",
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .copy()
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

expected_shape_628hc3a3a3a3 = (315, 8367)

for arr_628hc3a3a3a3 in (
    u_628hc3a3a3a3,
    level_628hc3a3a3a3,
    noise_628hc3a3a3a3,
    stuck_628hc3a3a3a3,
):
    if arr_628hc3a3a3a3.shape != expected_shape_628hc3a3a3a3:
        raise RuntimeError(
            f"Unexpected corrupted matrix shape: {arr_628hc3a3a3a3.shape}"
        )

for arr_628hc3a3a3a3 in (
    clean_u_628hc3a3a3a3,
    clean_level_628hc3a3a3a3,
    clean_noise_628hc3a3a3a3,
    clean_stuck_628hc3a3a3a3,
):
    if arr_628hc3a3a3a3.shape != (8367,):
        raise RuntimeError(
            f"Unexpected clean vector shape: {arr_628hc3a3a3a3.shape}"
        )

if not np.array_equal(
    manifest_628hc3a3a3a3["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(1, 316, dtype=np.int64),
):
    raise RuntimeError(
        "Manifest scenario order is not exactly 1..315."
    )

if not np.array_equal(
    utility_628hc3a3a3a3["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(1, 316, dtype=np.int64),
):
    raise RuntimeError(
        "Utility scenario order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 2. Mechanism-pattern scores
# ---------------------------------------------------------------

ns_628hc3a3a3a3 = np.minimum(
    noise_628hc3a3a3a3,
    stuck_628hc3a3a3a3,
)

lns_628hc3a3a3a3 = np.minimum.reduce(
    [
        level_628hc3a3a3a3,
        noise_628hc3a3a3a3,
        stuck_628hc3a3a3a3,
    ]
)

clean_ns_628hc3a3a3a3 = np.minimum(
    clean_noise_628hc3a3a3a3,
    clean_stuck_628hc3a3a3a3,
)

clean_lns_628hc3a3a3a3 = np.minimum.reduce(
    [
        clean_level_628hc3a3a3a3,
        clean_noise_628hc3a3a3a3,
        clean_stuck_628hc3a3a3a3,
    ]
)

# Positive disagreement means branch evidence is stronger
# than the model-uncertainty warning.

ns_gap_628hc3a3a3a3 = np.clip(
    ns_628hc3a3a3a3
    -
    u_628hc3a3a3a3,
    0.0,
    1.0,
)

lns_gap_628hc3a3a3a3 = np.clip(
    lns_628hc3a3a3a3
    -
    u_628hc3a3a3a3,
    0.0,
    1.0,
)

clean_ns_gap_628hc3a3a3a3 = np.clip(
    clean_ns_628hc3a3a3a3
    -
    clean_u_628hc3a3a3a3,
    0.0,
    1.0,
)

clean_lns_gap_628hc3a3a3a3 = np.clip(
    clean_lns_628hc3a3a3a3
    -
    clean_u_628hc3a3a3a3,
    0.0,
    1.0,
)

EPS_628HC3A3A3A3 = 1e-12

ns_relative_gap_628hc3a3a3a3 = np.clip(
    (
        ns_628hc3a3a3a3
        -
        u_628hc3a3a3a3
    )
    /
    (
        1.0
        -
        u_628hc3a3a3a3
        +
        EPS_628HC3A3A3A3
    ),
    0.0,
    1.0,
)

lns_relative_gap_628hc3a3a3a3 = np.clip(
    (
        lns_628hc3a3a3a3
        -
        u_628hc3a3a3a3
    )
    /
    (
        1.0
        -
        u_628hc3a3a3a3
        +
        EPS_628HC3A3A3A3
    ),
    0.0,
    1.0,
)

clean_ns_relative_gap_628hc3a3a3a3 = np.clip(
    (
        clean_ns_628hc3a3a3a3
        -
        clean_u_628hc3a3a3a3
    )
    /
    (
        1.0
        -
        clean_u_628hc3a3a3a3
        +
        EPS_628HC3A3A3A3
    ),
    0.0,
    1.0,
)

clean_lns_relative_gap_628hc3a3a3a3 = np.clip(
    (
        clean_lns_628hc3a3a3a3
        -
        clean_u_628hc3a3a3a3
    )
    /
    (
        1.0
        -
        clean_u_628hc3a3a3a3
        +
        EPS_628HC3A3A3A3
    ),
    0.0,
    1.0,
)

corrupted_scores_628hc3a3a3a3 = {
    "noise_stuck_gap":
        ns_gap_628hc3a3a3a3,

    "level_noise_stuck_gap":
        lns_gap_628hc3a3a3a3,

    "noise_stuck_relative_gap":
        ns_relative_gap_628hc3a3a3a3,

    "level_noise_stuck_relative_gap":
        lns_relative_gap_628hc3a3a3a3,
}

clean_scores_628hc3a3a3a3 = {
    "noise_stuck_gap":
        clean_ns_gap_628hc3a3a3a3,

    "level_noise_stuck_gap":
        clean_lns_gap_628hc3a3a3a3,

    "noise_stuck_relative_gap":
        clean_ns_relative_gap_628hc3a3a3a3,

    "level_noise_stuck_relative_gap":
        clean_lns_relative_gap_628hc3a3a3a3,
}

# ---------------------------------------------------------------
# 3. Clean DEV disagreement baseline
# ---------------------------------------------------------------

clean_rows_628hc3a3a3a3 = []

for (
    name_628hc3a3a3a3,
    values_628hc3a3a3a3,
) in clean_scores_628hc3a3a3a3.items():

    clean_rows_628hc3a3a3a3.append(
        {
            "score":
                name_628hc3a3a3a3,

            "clean_median":
                float(
                    np.median(
                        values_628hc3a3a3a3
                    )
                ),

            "clean_mean":
                float(
                    np.mean(
                        values_628hc3a3a3a3
                    )
                ),

            "clean_q90":
                float(
                    np.quantile(
                        values_628hc3a3a3a3,
                        0.90,
                    )
                ),

            "clean_q95":
                float(
                    np.quantile(
                        values_628hc3a3a3a3,
                        0.95,
                    )
                ),
        }
    )

FD002_C3A3A3A3_CLEAN_DISAGREEMENT = (
    pd.DataFrame(
        clean_rows_628hc3a3a3a3
    )
)

print(
    "\n--- CLEAN DEV DISAGREEMENT BASELINE ---"
)

display(
    FD002_C3A3A3A3_CLEAN_DISAGREEMENT
)

# ---------------------------------------------------------------
# 4. Scenario-level continuous summaries
# ---------------------------------------------------------------

scenario_df_628hc3a3a3a3 = (
    manifest_628hc3a3a3a3[
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .copy()
)

for (
    name_628hc3a3a3a3,
    matrix_628hc3a3a3a3,
) in corrupted_scores_628hc3a3a3a3.items():

    scenario_df_628hc3a3a3a3[
        f"{name_628hc3a3a3a3}_median"
    ] = np.median(
        matrix_628hc3a3a3a3,
        axis=1,
    )

    scenario_df_628hc3a3a3a3[
        f"{name_628hc3a3a3a3}_mean"
    ] = np.mean(
        matrix_628hc3a3a3a3,
        axis=1,
    )

    scenario_df_628hc3a3a3a3[
        f"{name_628hc3a3a3a3}_q90"
    ] = np.quantile(
        matrix_628hc3a3a3a3,
        0.90,
        axis=1,
    )

scenario_df_628hc3a3a3a3 = (
    scenario_df_628hc3a3a3a3
    .merge(
        utility_628hc3a3a3a3,
        on="scenario_number",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

FD002_C3A3A3A3_SCENARIO_DISAGREEMENT = (
    scenario_df_628hc3a3a3a3
)

# ---------------------------------------------------------------
# 5. Association with observed fusion utility
# ---------------------------------------------------------------

association_rows_628hc3a3a3a3 = []

for name_628hc3a3a3a3 in (
    corrupted_scores_628hc3a3a3a3.keys()
):

    for summary_628hc3a3a3a3 in (
        "median",
        "mean",
        "q90",
    ):

        column_628hc3a3a3a3 = (
            f"{name_628hc3a3a3a3}_"
            f"{summary_628hc3a3a3a3}"
        )

        rho_628hc3a3a3a3 = float(
            scenario_df_628hc3a3a3a3[
                [
                    column_628hc3a3a3a3,
                    "delta_gain80_vs_uncertainty",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1,
            ]
        )

        association_rows_628hc3a3a3a3.append(
            {
                "score":
                    name_628hc3a3a3a3,

                "summary":
                    summary_628hc3a3a3a3,

                "spearman_with_delta_gain80":
                    rho_628hc3a3a3a3,
            }
        )

FD002_C3A3A3A3_UTILITY_ASSOCIATION = (
    pd.DataFrame(
        association_rows_628hc3a3a3a3
    )
    .sort_values(
        "spearman_with_delta_gain80",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "\n--- DISAGREEMENT SCORE VS FUSION UTILITY ---"
)

display(
    FD002_C3A3A3A3_UTILITY_ASSOCIATION
)

# ---------------------------------------------------------------
# 6. Family × severity response
# ---------------------------------------------------------------

summary_columns_628hc3a3a3a3 = [
    column_628hc3a3a3a3
    for column_628hc3a3a3a3
    in scenario_df_628hc3a3a3a3.columns
    if (
        column_628hc3a3a3a3.endswith(
            "_median"
        )
        or
        column_628hc3a3a3a3.endswith(
            "_mean"
        )
        or
        column_628hc3a3a3a3.endswith(
            "_q90"
        )
    )
]

family_severity_628hc3a3a3a3 = (
    scenario_df_628hc3a3a3a3
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )[
        summary_columns_628hc3a3a3a3
        +
        [
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .median()
)

severity_order_628hc3a3a3a3 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_severity_628hc3a3a3a3[
    "_severity_order"
] = (
    family_severity_628hc3a3a3a3[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a3
    )
)

family_severity_628hc3a3a3a3 = (
    family_severity_628hc3a3a3a3
    .sort_values(
        [
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(drop=True)
)

FD002_C3A3A3A3_FAMILY_SEVERITY = (
    family_severity_628hc3a3a3a3
)

print(
    "\n--- FAMILY × SEVERITY DISAGREEMENT RESPONSE ---"
)

display(
    FD002_C3A3A3A3_FAMILY_SEVERITY
)

# ---------------------------------------------------------------
# 7. Strongest disagreement scenarios
# ---------------------------------------------------------------

top_score_name_628hc3a3a3a3 = str(
    FD002_C3A3A3A3_UTILITY_ASSOCIATION
    .iloc[0][
        "score"
    ]
)

top_summary_name_628hc3a3a3a3 = str(
    FD002_C3A3A3A3_UTILITY_ASSOCIATION
    .iloc[0][
        "summary"
    ]
)

top_column_628hc3a3a3a3 = (
    f"{top_score_name_628hc3a3a3a3}_"
    f"{top_summary_name_628hc3a3a3a3}"
)

FD002_C3A3A3A3_TOP_DISAGREEMENT = (
    scenario_df_628hc3a3a3a3
    .sort_values(
        [
            top_column_628hc3a3a3a3,
            "delta_gain80_vs_uncertainty",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .head(25)
    [
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            top_column_628hc3a3a3a3,
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\n--- TOP 25 SCENARIOS BY BEST "
    "DISAGREEMENT SUMMARY ---"
)

print(
    "Selected only for diagnostic display:",
    top_column_628hc3a3a3a3,
)

display(
    FD002_C3A3A3A3_TOP_DISAGREEMENT
)

# ---------------------------------------------------------------
# 8. Integrity audit
# ---------------------------------------------------------------

all_corrupted_scores_valid_628hc3a3a3a3 = bool(
    all(
        np.isfinite(
            matrix_628hc3a3a3a3
        ).all()
        and
        np.all(
            (
                matrix_628hc3a3a3a3
                >=
                0.0
            )
            &
            (
                matrix_628hc3a3a3a3
                <=
                1.0
            )
        )
        for matrix_628hc3a3a3a3
        in corrupted_scores_628hc3a3a3a3.values()
    )
)

all_clean_scores_valid_628hc3a3a3a3 = bool(
    all(
        np.isfinite(
            values_628hc3a3a3a3
        ).all()
        and
        np.all(
            (
                values_628hc3a3a3a3
                >=
                0.0
            )
            &
            (
                values_628hc3a3a3a3
                <=
                1.0
            )
        )
        for values_628hc3a3a3a3
        in clean_scores_628hc3a3a3a3.values()
    )
)

audit_628hc3a3a3a3 = pd.DataFrame(
    {
        "check": [
            "corrupted_shapes_exact",
            "clean_shapes_exact",
            "manifest_rows_315",
            "utility_rows_315",
            "scenario_summary_rows_315",
            "four_disagreement_scores",
            "twelve_association_rows",
            "family_severity_groups_15",
            "corrupted_scores_valid",
            "clean_scores_valid",
        ],

        "passed": [

            all(
                arr_628hc3a3a3a3.shape
                ==
                expected_shape_628hc3a3a3a3
                for arr_628hc3a3a3a3
                in (
                    u_628hc3a3a3a3,
                    level_628hc3a3a3a3,
                    noise_628hc3a3a3a3,
                    stuck_628hc3a3a3a3,
                )
            ),

            all(
                arr_628hc3a3a3a3.shape
                ==
                (8367,)
                for arr_628hc3a3a3a3
                in (
                    clean_u_628hc3a3a3a3,
                    clean_level_628hc3a3a3a3,
                    clean_noise_628hc3a3a3a3,
                    clean_stuck_628hc3a3a3a3,
                )
            ),

            len(
                manifest_628hc3a3a3a3
            )
            ==
            315,

            len(
                utility_628hc3a3a3a3
            )
            ==
            315,

            len(
                FD002_C3A3A3A3_SCENARIO_DISAGREEMENT
            )
            ==
            315,

            len(
                corrupted_scores_628hc3a3a3a3
            )
            ==
            4,

            len(
                FD002_C3A3A3A3_UTILITY_ASSOCIATION
            )
            ==
            12,

            len(
                FD002_C3A3A3A3_FAMILY_SEVERITY
            )
            ==
            15,

            all_corrupted_scores_valid_628hc3a3a3a3,

            all_clean_scores_valid_628hc3a3a3a3,
        ],
    }
)

print(
    "\n--- C3-A3-A3-A3 AUDIT ---"
)

display(
    audit_628hc3a3a3a3
)

all_checks_passed_628hc3a3a3a3 = bool(
    audit_628hc3a3a3a3[
        "passed"
    ]
    .all()
)

if not all_checks_passed_628hc3a3a3a3:
    raise RuntimeError(
        "C3-A3-A3-A3 disagreement audit failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

best_row_628hc3a3a3a3 = (
    FD002_C3A3A3A3_UTILITY_ASSOCIATION
    .iloc[0]
)

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A3 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Best diagnostic disagreement score :",
    best_row_628hc3a3a3a3[
        "score"
    ],
)

print(
    "Best diagnostic summary            :",
    best_row_628hc3a3a3a3[
        "summary"
    ],
)

print(
    "Best utility Spearman               :",
    float(
        best_row_628hc3a3a3a3[
            "spearman_with_delta_gain80"
        ]
    ),
)

print(
    "Scenarios evaluated                  :",
    315,
)

print(
    "All checks passed                    :",
    all_checks_passed_628hc3a3a3a3,
)

print("DEV RUL labels used                  : True")
print("Official TEST labels read            : False")
print("TEST scored                          : False")
print("Adaptive fusion rule selected        : False")
print("Combination weight selected          : False")
print("Abstention threshold selected        : False")
print("Model executed                       : False")
print("Model retrained                      : False")
print("Files written                        : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A3 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A4
# CONTINUOUS DISAGREEMENT-GATED FUSION DIAGNOSTIC
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A4 — CONTINUOUS DISAGREEMENT-GATED FUSION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Frozen aligned development objects
# ---------------------------------------------------------------

manifest_628hc3a3a3a4 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

prediction_628hc3a3a3a4 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64,
)

uncertainty_628hc3a3a3a4 = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

quality_628hc3a3a3a4 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

level_628hc3a3a3a4 = np.asarray(
    FD002_LEVEL_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

noise_628hc3a3a3a4 = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3a3a4 = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

y_dev_628hc3a3a3a4 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64,
).reshape(-1)

clean_prediction_628hc3a3a3a4 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
).reshape(-1)

clean_uncertainty_628hc3a3a3a4 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_quality_container_628hc3a3a3a4 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

clean_quality_628hc3a3a3a4 = np.asarray(
    clean_quality_container_628hc3a3a3a4[
        "sensor_quality_risk"
    ],
    dtype=np.float64,
).reshape(-1)

clean_level_628hc3a3a3a4 = np.asarray(
    clean_quality_container_628hc3a3a3a4[
        "level"
    ],
    dtype=np.float64,
).reshape(-1)

clean_noise_628hc3a3a3a4 = np.asarray(
    clean_quality_container_628hc3a3a3a4[
        "noise"
    ],
    dtype=np.float64,
).reshape(-1)

clean_stuck_628hc3a3a3a4 = np.asarray(
    clean_quality_container_628hc3a3a3a4[
        "stuck"
    ],
    dtype=np.float64,
).reshape(-1)

EXPECTED_SHAPE_628HC3A3A3A4 = (
    315,
    8367,
)

for matrix_628hc3a3a3a4 in (
    prediction_628hc3a3a3a4,
    uncertainty_628hc3a3a3a4,
    quality_628hc3a3a3a4,
    level_628hc3a3a3a4,
    noise_628hc3a3a3a4,
    stuck_628hc3a3a3a4,
):
    if (
        matrix_628hc3a3a3a4.shape
        !=
        EXPECTED_SHAPE_628HC3A3A3A4
    ):
        raise RuntimeError(
            "Unexpected corrupted matrix shape: "
            f"{matrix_628hc3a3a3a4.shape}"
        )

for vector_628hc3a3a3a4 in (
    y_dev_628hc3a3a3a4,
    clean_prediction_628hc3a3a3a4,
    clean_uncertainty_628hc3a3a3a4,
    clean_quality_628hc3a3a3a4,
    clean_level_628hc3a3a3a4,
    clean_noise_628hc3a3a3a4,
    clean_stuck_628hc3a3a3a4,
):
    if vector_628hc3a3a3a4.shape != (8367,):
        raise RuntimeError(
            "Unexpected clean vector shape: "
            f"{vector_628hc3a3a3a4.shape}"
        )

if not np.array_equal(
    manifest_628hc3a3a3a4[
        "scenario_number"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Scenario order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 2. Fixed diagnostic contract
#
# Gamma remains the previously diagnosed C3-A2 value.
# It is NOT selected/frozen here.
# ---------------------------------------------------------------

GAMMA_628HC3A3A3A4 = 0.80

COVERAGES_628HC3A3A3A4 = (
    0.90,
    0.80,
    0.70,
)

QUALITY_Q95_628HC3A3A3A4 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)

EPS_628HC3A3A3A4 = 1e-12

# ---------------------------------------------------------------
# 3. Continuous disagreement gates
# ---------------------------------------------------------------

noise_stuck_628hc3a3a3a4 = np.minimum(
    noise_628hc3a3a3a4,
    stuck_628hc3a3a3a4,
)

level_noise_stuck_628hc3a3a3a4 = (
    np.minimum.reduce(
        [
            level_628hc3a3a3a4,
            noise_628hc3a3a3a4,
            stuck_628hc3a3a3a4,
        ]
    )
)

clean_noise_stuck_628hc3a3a3a4 = np.minimum(
    clean_noise_628hc3a3a3a4,
    clean_stuck_628hc3a3a3a4,
)

clean_level_noise_stuck_628hc3a3a3a4 = (
    np.minimum.reduce(
        [
            clean_level_628hc3a3a3a4,
            clean_noise_628hc3a3a3a4,
            clean_stuck_628hc3a3a3a4,
        ]
    )
)

ns_gap_628hc3a3a3a4 = np.clip(
    noise_stuck_628hc3a3a3a4
    -
    uncertainty_628hc3a3a3a4,
    0.0,
    1.0,
)

lns_gap_628hc3a3a3a4 = np.clip(
    level_noise_stuck_628hc3a3a3a4
    -
    uncertainty_628hc3a3a3a4,
    0.0,
    1.0,
)

ns_relative_gap_628hc3a3a3a4 = np.clip(
    (
        noise_stuck_628hc3a3a3a4
        -
        uncertainty_628hc3a3a3a4
    )
    /
    (
        1.0
        -
        uncertainty_628hc3a3a3a4
        +
        EPS_628HC3A3A3A4
    ),
    0.0,
    1.0,
)

lns_relative_gap_628hc3a3a3a4 = np.clip(
    (
        level_noise_stuck_628hc3a3a3a4
        -
        uncertainty_628hc3a3a3a4
    )
    /
    (
        1.0
        -
        uncertainty_628hc3a3a3a4
        +
        EPS_628HC3A3A3A4
    ),
    0.0,
    1.0,
)

clean_ns_gap_628hc3a3a3a4 = np.clip(
    clean_noise_stuck_628hc3a3a3a4
    -
    clean_uncertainty_628hc3a3a3a4,
    0.0,
    1.0,
)

clean_lns_gap_628hc3a3a3a4 = np.clip(
    clean_level_noise_stuck_628hc3a3a3a4
    -
    clean_uncertainty_628hc3a3a3a4,
    0.0,
    1.0,
)

clean_ns_relative_gap_628hc3a3a3a4 = np.clip(
    (
        clean_noise_stuck_628hc3a3a3a4
        -
        clean_uncertainty_628hc3a3a3a4
    )
    /
    (
        1.0
        -
        clean_uncertainty_628hc3a3a3a4
        +
        EPS_628HC3A3A3A4
    ),
    0.0,
    1.0,
)

clean_lns_relative_gap_628hc3a3a3a4 = np.clip(
    (
        clean_level_noise_stuck_628hc3a3a3a4
        -
        clean_uncertainty_628hc3a3a3a4
    )
    /
    (
        1.0
        -
        clean_uncertainty_628hc3a3a3a4
        +
        EPS_628HC3A3A3A4
    ),
    0.0,
    1.0,
)

# ---------------------------------------------------------------
# 4. Candidate architecture set
#
# No new threshold is introduced.
# ---------------------------------------------------------------

CANDIDATES_628HC3A3A3A4 = (
    "uncertainty_only",
    "fixed_global_g080",
    "noise_stuck_gap_g080",
    "noise_stuck_relative_gap_g080",
    "level_noise_stuck_gap_g080",
    "level_noise_stuck_relative_gap_g080",
)

print(
    "\n--- CANDIDATE ARCHITECTURES ---"
)

for candidate_628hc3a3a3a4 in (
    CANDIDATES_628HC3A3A3A4
):
    print(
        candidate_628hc3a3a3a4
    )

# ---------------------------------------------------------------
# 5. Helpers
# ---------------------------------------------------------------

def quality_excess_local_628hc3a3a3a4(
    quality_values
):
    quality_values = np.asarray(
        quality_values,
        dtype=np.float64,
    )

    denominator = (
        1.0
        -
        QUALITY_Q95_628HC3A3A3A4
    )

    if denominator <= 0.0:
        raise RuntimeError(
            "Invalid FIT quality q95 marker."
        )

    return np.clip(
        (
            quality_values
            -
            QUALITY_Q95_628HC3A3A3A4
        )
        /
        denominator,
        0.0,
        1.0,
    )


def fuse_with_gate_628hc3a3a3a4(
    uncertainty_values,
    quality_values,
    gate_values,
    gamma,
):
    uncertainty_values = np.asarray(
        uncertainty_values,
        dtype=np.float64,
    )

    quality_values = np.asarray(
        quality_values,
        dtype=np.float64,
    )

    gate_values = np.asarray(
        gate_values,
        dtype=np.float64,
    )

    if (
        uncertainty_values.shape
        !=
        quality_values.shape
        or
        uncertainty_values.shape
        !=
        gate_values.shape
    ):
        raise ValueError(
            "U, Q, and gate arrays must have identical shapes."
        )

    q_excess = (
        quality_excess_local_628hc3a3a3a4(
            quality_values
        )
    )

    combined = (
        uncertainty_values
        +
        (
            1.0
            -
            uncertainty_values
        )
        *
        float(gamma)
        *
        gate_values
        *
        q_excess
    )

    return np.clip(
        combined,
        0.0,
        1.0,
    )


def candidate_risk_628hc3a3a3a4(
    candidate,
    uncertainty_values,
    quality_values,
    ns_gap_values,
    ns_relative_gap_values,
    lns_gap_values,
    lns_relative_gap_values,
):
    if candidate == "uncertainty_only":
        return np.asarray(
            uncertainty_values,
            dtype=np.float64,
        ).copy()

    if candidate == "fixed_global_g080":
        gate_values = np.ones_like(
            uncertainty_values,
            dtype=np.float64,
        )

    elif candidate == "noise_stuck_gap_g080":
        gate_values = ns_gap_values

    elif (
        candidate
        ==
        "noise_stuck_relative_gap_g080"
    ):
        gate_values = ns_relative_gap_values

    elif (
        candidate
        ==
        "level_noise_stuck_gap_g080"
    ):
        gate_values = lns_gap_values

    elif (
        candidate
        ==
        "level_noise_stuck_relative_gap_g080"
    ):
        gate_values = lns_relative_gap_values

    else:
        raise ValueError(
            f"Unknown candidate: {candidate}"
        )

    return fuse_with_gate_628hc3a3a3a4(
        uncertainty_values,
        quality_values,
        gate_values,
        GAMMA_628HC3A3A3A4,
    )


def evaluate_coverages_628hc3a3a3a4(
    risk_values,
    absolute_error
):
    full_mae = float(
        np.mean(
            absolute_error
        )
    )

    output = {
        "full_mae":
            full_mae,
    }

    for coverage_628hc3a3a3a4 in (
        COVERAGES_628HC3A3A3A4
    ):
        evaluated = (
            selective_mae_tie_aware_628hc3a2(
                risk_values,
                absolute_error,
                coverage_628hc3a3a3a4,
            )
        )

        pct_628hc3a3a3a4 = int(
            round(
                100
                *
                coverage_628hc3a3a3a4
            )
        )

        mae_628hc3a3a3a4 = float(
            evaluated[
                "mae"
            ]
        )

        output[
            f"mae_{pct_628hc3a3a3a4}"
        ] = mae_628hc3a3a3a4

        output[
            f"gain_{pct_628hc3a3a3a4}"
        ] = float(
            full_mae
            -
            mae_628hc3a3a3a4
        )

    return output

# ---------------------------------------------------------------
# 6. Clean DEV candidate response
# ---------------------------------------------------------------

clean_error_628hc3a3a3a4 = np.abs(
    clean_prediction_628hc3a3a3a4
    -
    y_dev_628hc3a3a3a4
)

clean_rows_628hc3a3a3a4 = []

for candidate_628hc3a3a3a4 in (
    CANDIDATES_628HC3A3A3A4
):

    risk_628hc3a3a3a4 = (
        candidate_risk_628hc3a3a3a4(
            candidate_628hc3a3a3a4,
            clean_uncertainty_628hc3a3a3a4,
            clean_quality_628hc3a3a3a4,
            clean_ns_gap_628hc3a3a3a4,
            clean_ns_relative_gap_628hc3a3a3a4,
            clean_lns_gap_628hc3a3a3a4,
            clean_lns_relative_gap_628hc3a3a3a4,
        )
    )

    metrics_628hc3a3a3a4 = (
        evaluate_coverages_628hc3a3a3a4(
            risk_628hc3a3a3a4,
            clean_error_628hc3a3a3a4,
        )
    )

    clean_rows_628hc3a3a3a4.append(
        {
            "candidate":
                candidate_628hc3a3a3a4,

            **metrics_628hc3a3a3a4,
        }
    )

FD002_C3A3A3A4_CLEAN_RESPONSE = (
    pd.DataFrame(
        clean_rows_628hc3a3a3a4
    )
)

print(
    "\n--- CLEAN DEV DISAGREEMENT-GATED RESPONSE ---"
)

display(
    FD002_C3A3A3A4_CLEAN_RESPONSE
)

# ---------------------------------------------------------------
# 7. Evaluate all 315 scenarios × six architectures
# ---------------------------------------------------------------

scenario_rows_628hc3a3a3a4 = []

for candidate_628hc3a3a3a4 in (
    CANDIDATES_628HC3A3A3A4
):

    for i_628hc3a3a3a4 in range(
        315
    ):

        risk_628hc3a3a3a4 = (
            candidate_risk_628hc3a3a3a4(
                candidate_628hc3a3a3a4,
                uncertainty_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
                quality_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
                ns_gap_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
                ns_relative_gap_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
                lns_gap_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
                lns_relative_gap_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ],
            )
        )

        absolute_error_628hc3a3a3a4 = (
            np.abs(
                prediction_628hc3a3a3a4[
                    i_628hc3a3a3a4
                ]
                -
                y_dev_628hc3a3a3a4
            )
        )

        metrics_628hc3a3a3a4 = (
            evaluate_coverages_628hc3a3a3a4(
                risk_628hc3a3a3a4,
                absolute_error_628hc3a3a3a4,
            )
        )

        manifest_row_628hc3a3a3a4 = (
            manifest_628hc3a3a3a4.iloc[
                i_628hc3a3a3a4
            ]
        )

        scenario_rows_628hc3a3a3a4.append(
            {
                "candidate":
                    candidate_628hc3a3a3a4,

                "scenario_id":
                    str(
                        manifest_row_628hc3a3a3a4[
                            "scenario_id"
                        ]
                    ),

                "scenario_number":
                    int(
                        manifest_row_628hc3a3a3a4[
                            "scenario_number"
                        ]
                    ),

                "family":
                    str(
                        manifest_row_628hc3a3a3a4[
                            "family"
                        ]
                    ),

                "sensor":
                    str(
                        manifest_row_628hc3a3a3a4[
                            "sensor"
                        ]
                    ),

                "severity":
                    str(
                        manifest_row_628hc3a3a3a4[
                            "severity"
                        ]
                    ),

                **metrics_628hc3a3a3a4,
            }
        )

FD002_C3A3A3A4_SCENARIO_METRICS = (
    pd.DataFrame(
        scenario_rows_628hc3a3a3a4
    )
)

if len(
    FD002_C3A3A3A4_SCENARIO_METRICS
) != (
    315
    *
    len(
        CANDIDATES_628HC3A3A3A4
    )
):
    raise RuntimeError(
        "Unexpected candidate-scenario row count."
    )

# ---------------------------------------------------------------
# 8. Overall architecture summary
# ---------------------------------------------------------------

baseline_628hc3a3a3a4 = (
    FD002_C3A3A3A4_SCENARIO_METRICS[
        FD002_C3A3A3A4_SCENARIO_METRICS[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ]
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

summary_rows_628hc3a3a3a4 = []

for candidate_628hc3a3a3a4 in (
    CANDIDATES_628HC3A3A3A4
):

    candidate_df_628hc3a3a3a4 = (
        FD002_C3A3A3A4_SCENARIO_METRICS[
            FD002_C3A3A3A4_SCENARIO_METRICS[
                "candidate"
            ]
            ==
            candidate_628hc3a3a3a4
        ]
        .sort_values(
            "scenario_number",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    row_628hc3a3a3a4 = {
        "candidate":
            candidate_628hc3a3a3a4,
    }

    for pct_628hc3a3a3a4 in (
        90,
        80,
        70,
    ):

        gain_column_628hc3a3a3a4 = (
            f"gain_{pct_628hc3a3a3a4}"
        )

        gains_628hc3a3a3a4 = (
            candidate_df_628hc3a3a3a4[
                gain_column_628hc3a3a3a4
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        baseline_gains_628hc3a3a3a4 = (
            baseline_628hc3a3a3a4[
                gain_column_628hc3a3a3a4
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        delta_628hc3a3a3a4 = (
            gains_628hc3a3a3a4
            -
            baseline_gains_628hc3a3a3a4
        )

        family_medians_628hc3a3a3a4 = (
            candidate_df_628hc3a3a3a4
            .groupby(
                "family",
                observed=True,
            )[
                gain_column_628hc3a3a3a4
            ]
            .median()
        )

        row_628hc3a3a3a4[
            f"median_gain_{pct_628hc3a3a3a4}"
        ] = float(
            np.median(
                gains_628hc3a3a3a4
            )
        )

        row_628hc3a3a3a4[
            f"mean_gain_{pct_628hc3a3a3a4}"
        ] = float(
            np.mean(
                gains_628hc3a3a3a4
            )
        )

        row_628hc3a3a3a4[
            f"q10_gain_{pct_628hc3a3a3a4}"
        ] = float(
            np.quantile(
                gains_628hc3a3a3a4,
                0.10,
            )
        )

        row_628hc3a3a3a4[
            f"worst_family_median_gain_{pct_628hc3a3a3a4}"
        ] = float(
            family_medians_628hc3a3a3a4.min()
        )

        row_628hc3a3a3a4[
            f"median_delta_vs_u_{pct_628hc3a3a3a4}"
        ] = float(
            np.median(
                delta_628hc3a3a3a4
            )
        )

        row_628hc3a3a3a4[
            f"mean_delta_vs_u_{pct_628hc3a3a3a4}"
        ] = float(
            np.mean(
                delta_628hc3a3a3a4
            )
        )

        row_628hc3a3a3a4[
            f"improved_vs_u_{pct_628hc3a3a3a4}"
        ] = int(
            np.sum(
                delta_628hc3a3a3a4
                >
                1e-12
            )
        )

        row_628hc3a3a3a4[
            f"equal_vs_u_{pct_628hc3a3a3a4}"
        ] = int(
            np.sum(
                np.abs(
                    delta_628hc3a3a3a4
                )
                <=
                1e-12
            )
        )

        row_628hc3a3a3a4[
            f"worse_vs_u_{pct_628hc3a3a3a4}"
        ] = int(
            np.sum(
                delta_628hc3a3a3a4
                <
                -1e-12
            )
        )

    summary_rows_628hc3a3a3a4.append(
        row_628hc3a3a3a4
    )

FD002_C3A3A3A4_ARCHITECTURE_SUMMARY = (
    pd.DataFrame(
        summary_rows_628hc3a3a3a4
    )
)

print(
    "\n--- DISAGREEMENT-GATED ARCHITECTURE SUMMARY ---"
)

display(
    FD002_C3A3A3A4_ARCHITECTURE_SUMMARY
)

# ---------------------------------------------------------------
# 9. Family × severity response at primary 80% coverage
# ---------------------------------------------------------------

family_severity_628hc3a3a3a4 = (
    FD002_C3A3A3A4_SCENARIO_METRICS
    .groupby(
        [
            "candidate",
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        n_scenarios=(
            "scenario_number",
            "size",
        ),

        median_gain80=(
            "gain_80",
            "median",
        ),

        mean_gain80=(
            "gain_80",
            "mean",
        ),
    )
)

severity_order_628hc3a3a3a4 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_severity_628hc3a3a3a4[
    "_severity_order"
] = (
    family_severity_628hc3a3a3a4[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a4
    )
)

FD002_C3A3A3A4_FAMILY_SEVERITY = (
    family_severity_628hc3a3a3a4
    .sort_values(
        [
            "candidate",
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(drop=True)
)

print(
    "\n--- FAMILY × SEVERITY RESPONSE @80% COVERAGE ---"
)

display(
    FD002_C3A3A3A4_FAMILY_SEVERITY
)

# ---------------------------------------------------------------
# 10. Stuck-only compact comparison
# ---------------------------------------------------------------

FD002_C3A3A3A4_STUCK_RESPONSE = (
    FD002_C3A3A3A4_FAMILY_SEVERITY[
        FD002_C3A3A3A4_FAMILY_SEVERITY[
            "family"
        ]
        ==
        "stuck"
    ]
    .reset_index(drop=True)
)

print(
    "\n--- STUCK-FAMILY DISAGREEMENT-GATED RESPONSE ---"
)

display(
    FD002_C3A3A3A4_STUCK_RESPONSE
)

# ---------------------------------------------------------------
# 11. Clean response
# ---------------------------------------------------------------

clean_display_628hc3a3a3a4 = (
    FD002_C3A3A3A4_CLEAN_RESPONSE[
        [
            "candidate",
            "gain_90",
            "gain_80",
            "gain_70",
        ]
    ]
    .copy()
)

print(
    "\n--- CLEAN DEV SELECTIVE GAINS ---"
)

display(
    clean_display_628hc3a3a3a4
)

# ---------------------------------------------------------------
# 12. Integrity audit
# ---------------------------------------------------------------

expected_rows_628hc3a3a3a4 = (
    315
    *
    len(
        CANDIDATES_628HC3A3A3A4
    )
)

metric_columns_628hc3a3a3a4 = [
    "full_mae",
    "mae_90",
    "gain_90",
    "mae_80",
    "gain_80",
    "mae_70",
    "gain_70",
]

audit_628hc3a3a3a4 = pd.DataFrame(
    {
        "check": [
            "candidate_count_6",
            "scenario_count_315",
            "dev_windows_8367",
            "scenario_candidate_rows_exact",
            "all_metrics_finite",
            "clean_candidate_rows_6",
            "three_coverages_evaluated",
            "scenario_order_exact",
        ],

        "passed": [

            len(
                CANDIDATES_628HC3A3A3A4
            )
            ==
            6,

            len(
                manifest_628hc3a3a3a4
            )
            ==
            315,

            len(
                y_dev_628hc3a3a3a4
            )
            ==
            8367,

            len(
                FD002_C3A3A3A4_SCENARIO_METRICS
            )
            ==
            expected_rows_628hc3a3a3a4,

            bool(
                np.isfinite(
                    FD002_C3A3A3A4_SCENARIO_METRICS[
                        metric_columns_628hc3a3a3a4
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                )
                .all()
            ),

            len(
                FD002_C3A3A3A4_CLEAN_RESPONSE
            )
            ==
            6,

            len(
                COVERAGES_628HC3A3A3A4
            )
            ==
            3,

            bool(
                np.array_equal(
                    manifest_628hc3a3a3a4[
                        "scenario_number"
                    ]
                    .to_numpy(
                        dtype=np.int64
                    ),
                    np.arange(
                        1,
                        316,
                        dtype=np.int64,
                    ),
                )
            ),
        ],
    }
)

print(
    "\n--- C3-A3-A3-A4 AUDIT ---"
)

display(
    audit_628hc3a3a3a4
)

all_checks_passed_628hc3a3a3a4 = bool(
    audit_628hc3a3a3a4[
        "passed"
    ]
    .all()
)

if not all_checks_passed_628hc3a3a3a4:
    raise RuntimeError(
        "C3-A3-A3-A4 diagnostic audit failed."
    )

# ---------------------------------------------------------------
# 13. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A4 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Candidates evaluated               :",
    len(
        CANDIDATES_628HC3A3A3A4
    ),
)

print(
    "Scenario-candidate evaluations     :",
    expected_rows_628hc3a3a3a4,
)

print(
    "Diagnostic gamma                   :",
    GAMMA_628HC3A3A3A4,
)

print(
    "Coverages evaluated                :",
    COVERAGES_628HC3A3A3A4,
)

print(
    "New disagreement threshold selected:",
    False,
)

print(
    "All checks passed                  :",
    all_checks_passed_628hc3a3a3a4,
)

print("DEV RUL labels used                : True")
print("Official TEST labels read          : False")
print("TEST scored                        : False")
print("Adaptive fusion rule selected      : False")
print("Combination weight selected        : False")
print("Abstention threshold selected      : False")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files written                      : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A4 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A5
# CAUSAL PERSISTENCE-OF-DISAGREEMENT AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A5 — CAUSAL PERSISTENCE-OF-DISAGREEMENT AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Frozen aligned inputs
# ---------------------------------------------------------------

manifest_628hc3a3a3a5 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

dev_keys_628hc3a3a3a5 = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)

uncertainty_628hc3a3a3a5 = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

noise_628hc3a3a3a5 = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3a3a5 = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

clean_uncertainty_628hc3a3a3a5 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_branch_dict_628hc3a3a3a5 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

clean_noise_628hc3a3a3a5 = np.asarray(
    clean_branch_dict_628hc3a3a3a5[
        "noise"
    ],
    dtype=np.float64,
).reshape(-1)

clean_stuck_628hc3a3a3a5 = np.asarray(
    clean_branch_dict_628hc3a3a3a5[
        "stuck"
    ],
    dtype=np.float64,
).reshape(-1)

utility_628hc3a3a3a5 = (
    FD002_C3A3A3A1_ALIGNMENT_628HC3A3A3A1[
        [
            "scenario_number",
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

EXPECTED_SHAPE_628HC3A3A3A5 = (
    315,
    8367,
)

for matrix_628hc3a3a3a5 in (
    uncertainty_628hc3a3a3a5,
    noise_628hc3a3a3a5,
    stuck_628hc3a3a3a5,
):

    if (
        matrix_628hc3a3a3a5.shape
        !=
        EXPECTED_SHAPE_628HC3A3A3A5
    ):
        raise RuntimeError(
            "Unexpected corrupted matrix shape: "
            f"{matrix_628hc3a3a3a5.shape}"
        )

for vector_628hc3a3a3a5 in (
    clean_uncertainty_628hc3a3a3a5,
    clean_noise_628hc3a3a3a5,
    clean_stuck_628hc3a3a3a5,
):

    if (
        vector_628hc3a3a3a5.shape
        !=
        (8367,)
    ):
        raise RuntimeError(
            "Unexpected clean vector shape: "
            f"{vector_628hc3a3a3a5.shape}"
        )

required_key_columns_628hc3a3a3a5 = {
    "unit_id",
    "cycle",
}

if not (
    required_key_columns_628hc3a3a3a5
    .issubset(
        dev_keys_628hc3a3a3a5.columns
    )
):
    raise RuntimeError(
        "DEV window keys are missing unit_id or cycle."
    )

if len(
    dev_keys_628hc3a3a3a5
) != 8367:
    raise RuntimeError(
        "DEV window-key count is not 8367."
    )

if (
    dev_keys_628hc3a3a3a5[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "Duplicate DEV window keys detected."
    )

if not np.array_equal(
    manifest_628hc3a3a3a5[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Scenario order is not exactly 1..315."
    )

if not np.array_equal(
    utility_628hc3a3a3a5[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Utility order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 2. Raw disagreement evidence
# ---------------------------------------------------------------

noise_stuck_628hc3a3a3a5 = np.minimum(
    noise_628hc3a3a3a5,
    stuck_628hc3a3a3a5,
)

clean_noise_stuck_628hc3a3a3a5 = np.minimum(
    clean_noise_628hc3a3a3a5,
    clean_stuck_628hc3a3a3a5,
)

gap_628hc3a3a3a5 = np.clip(
    noise_stuck_628hc3a3a3a5
    -
    uncertainty_628hc3a3a3a5,
    0.0,
    1.0,
)

clean_gap_628hc3a3a3a5 = np.clip(
    clean_noise_stuck_628hc3a3a3a5
    -
    clean_uncertainty_628hc3a3a3a5,
    0.0,
    1.0,
)

positive_evidence_628hc3a3a3a5 = (
    gap_628hc3a3a3a5
    >
    0.0
).astype(
    np.float64
)

clean_positive_evidence_628hc3a3a3a5 = (
    clean_gap_628hc3a3a3a5
    >
    0.0
).astype(
    np.float64
)

# ---------------------------------------------------------------
# 3. Causal within-engine persistence
#
# Only current and previous windows from the same engine are used.
# ---------------------------------------------------------------

MAJORITY_LEVEL_628HC3A3A3A5 = 0.50

causal_fraction_628hc3a3a3a5 = np.zeros_like(
    gap_628hc3a3a3a5,
    dtype=np.float64,
)

clean_causal_fraction_628hc3a3a3a5 = np.zeros(
    8367,
    dtype=np.float64,
)

unit_ids_628hc3a3a3a5 = (
    dev_keys_628hc3a3a3a5[
        "unit_id"
    ]
    .to_numpy()
)

cycles_628hc3a3a3a5 = (
    dev_keys_628hc3a3a3a5[
        "cycle"
    ]
    .to_numpy()
)

unique_units_628hc3a3a3a5 = np.unique(
    unit_ids_628hc3a3a3a5
)

for unit_id_628hc3a3a3a5 in (
    unique_units_628hc3a3a3a5
):

    unit_positions_628hc3a3a3a5 = (
        np.flatnonzero(
            unit_ids_628hc3a3a3a5
            ==
            unit_id_628hc3a3a3a5
        )
    )

    unit_cycles_628hc3a3a3a5 = (
        cycles_628hc3a3a3a5[
            unit_positions_628hc3a3a3a5
        ]
    )

    order_628hc3a3a3a5 = np.argsort(
        unit_cycles_628hc3a3a3a5,
        kind="mergesort",
    )

    ordered_positions_628hc3a3a3a5 = (
        unit_positions_628hc3a3a3a5[
            order_628hc3a3a3a5
        ]
    )

    n_unit_windows_628hc3a3a3a5 = len(
        ordered_positions_628hc3a3a3a5
    )

    denominators_628hc3a3a3a5 = np.arange(
        1,
        n_unit_windows_628hc3a3a3a5 + 1,
        dtype=np.float64,
    )

    corrupted_unit_evidence_628hc3a3a3a5 = (
        positive_evidence_628hc3a3a3a5[
            :,
            ordered_positions_628hc3a3a3a5,
        ]
    )

    corrupted_unit_fraction_628hc3a3a3a5 = (
        np.cumsum(
            corrupted_unit_evidence_628hc3a3a3a5,
            axis=1,
        )
        /
        denominators_628hc3a3a3a5[
            None,
            :
        ]
    )

    causal_fraction_628hc3a3a3a5[
        :,
        ordered_positions_628hc3a3a3a5,
    ] = corrupted_unit_fraction_628hc3a3a3a5

    clean_unit_evidence_628hc3a3a3a5 = (
        clean_positive_evidence_628hc3a3a3a5[
            ordered_positions_628hc3a3a3a5
        ]
    )

    clean_unit_fraction_628hc3a3a3a5 = (
        np.cumsum(
            clean_unit_evidence_628hc3a3a3a5
        )
        /
        denominators_628hc3a3a3a5
    )

    clean_causal_fraction_628hc3a3a3a5[
        ordered_positions_628hc3a3a3a5
    ] = clean_unit_fraction_628hc3a3a3a5

majority_excess_628hc3a3a3a5 = np.clip(
    (
        causal_fraction_628hc3a3a3a5
        -
        MAJORITY_LEVEL_628HC3A3A3A5
    )
    /
    (
        1.0
        -
        MAJORITY_LEVEL_628HC3A3A3A5
    ),
    0.0,
    1.0,
)

clean_majority_excess_628hc3a3a3a5 = np.clip(
    (
        clean_causal_fraction_628hc3a3a3a5
        -
        MAJORITY_LEVEL_628HC3A3A3A5
    )
    /
    (
        1.0
        -
        MAJORITY_LEVEL_628HC3A3A3A5
    ),
    0.0,
    1.0,
)

# ---------------------------------------------------------------
# 4. Clean DEV persistence baseline
# ---------------------------------------------------------------

FD002_C3A3A3A5_CLEAN_PERSISTENCE = (
    pd.DataFrame(
        [
            {
                "score":
                    "causal_positive_fraction",

                "median":
                    float(
                        np.median(
                            clean_causal_fraction_628hc3a3a3a5
                        )
                    ),

                "mean":
                    float(
                        np.mean(
                            clean_causal_fraction_628hc3a3a3a5
                        )
                    ),

                "q90":
                    float(
                        np.quantile(
                            clean_causal_fraction_628hc3a3a3a5,
                            0.90,
                        )
                    ),

                "q95":
                    float(
                        np.quantile(
                            clean_causal_fraction_628hc3a3a3a5,
                            0.95,
                        )
                    ),

                "nonzero_fraction":
                    float(
                        np.mean(
                            clean_causal_fraction_628hc3a3a3a5
                            >
                            0.0
                        )
                    ),
            },

            {
                "score":
                    "causal_majority_excess",

                "median":
                    float(
                        np.median(
                            clean_majority_excess_628hc3a3a3a5
                        )
                    ),

                "mean":
                    float(
                        np.mean(
                            clean_majority_excess_628hc3a3a3a5
                        )
                    ),

                "q90":
                    float(
                        np.quantile(
                            clean_majority_excess_628hc3a3a3a5,
                            0.90,
                        )
                    ),

                "q95":
                    float(
                        np.quantile(
                            clean_majority_excess_628hc3a3a3a5,
                            0.95,
                        )
                    ),

                "nonzero_fraction":
                    float(
                        np.mean(
                            clean_majority_excess_628hc3a3a3a5
                            >
                            0.0
                        )
                    ),
            },
        ]
    )
)

print(
    "\n--- CLEAN DEV CAUSAL-PERSISTENCE BASELINE ---"
)

display(
    FD002_C3A3A3A5_CLEAN_PERSISTENCE
)

# ---------------------------------------------------------------
# 5. Scenario-level persistence summaries
# ---------------------------------------------------------------

scenario_persistence_628hc3a3a3a5 = (
    manifest_628hc3a3a3a5[
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .copy()
)

scenario_persistence_628hc3a3a3a5[
    "causal_fraction_median"
] = np.median(
    causal_fraction_628hc3a3a3a5,
    axis=1,
)

scenario_persistence_628hc3a3a3a5[
    "causal_fraction_mean"
] = np.mean(
    causal_fraction_628hc3a3a3a5,
    axis=1,
)

scenario_persistence_628hc3a3a3a5[
    "majority_excess_median"
] = np.median(
    majority_excess_628hc3a3a3a5,
    axis=1,
)

scenario_persistence_628hc3a3a3a5[
    "majority_excess_mean"
] = np.mean(
    majority_excess_628hc3a3a3a5,
    axis=1,
)

scenario_persistence_628hc3a3a3a5[
    "majority_active_fraction"
] = np.mean(
    majority_excess_628hc3a3a3a5
    >
    0.0,
    axis=1,
)

scenario_persistence_628hc3a3a3a5 = (
    scenario_persistence_628hc3a3a3a5
    .merge(
        utility_628hc3a3a3a5,
        on="scenario_number",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

FD002_C3A3A3A5_SCENARIO_PERSISTENCE = (
    scenario_persistence_628hc3a3a3a5
)

# ---------------------------------------------------------------
# 6. Persistence association with fusion utility
# ---------------------------------------------------------------

association_columns_628hc3a3a3a5 = [
    "causal_fraction_median",
    "causal_fraction_mean",
    "majority_excess_median",
    "majority_excess_mean",
    "majority_active_fraction",
]

association_rows_628hc3a3a3a5 = []

for column_628hc3a3a3a5 in (
    association_columns_628hc3a3a3a5
):

    rho_628hc3a3a3a5 = float(
        scenario_persistence_628hc3a3a3a5[
            [
                column_628hc3a3a3a5,
                "delta_gain80_vs_uncertainty",
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1,
        ]
    )

    association_rows_628hc3a3a3a5.append(
        {
            "persistence_summary":
                column_628hc3a3a3a5,

            "spearman_with_delta_gain80":
                rho_628hc3a3a3a5,
        }
    )

FD002_C3A3A3A5_UTILITY_ASSOCIATION = (
    pd.DataFrame(
        association_rows_628hc3a3a3a5
    )
    .sort_values(
        "spearman_with_delta_gain80",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "\n--- CAUSAL PERSISTENCE VS FUSION UTILITY ---"
)

display(
    FD002_C3A3A3A5_UTILITY_ASSOCIATION
)

# ---------------------------------------------------------------
# 7. Family × severity persistence response
# ---------------------------------------------------------------

FD002_C3A3A3A5_FAMILY_SEVERITY = (
    scenario_persistence_628hc3a3a3a5
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )
    [
        [
            "causal_fraction_median",
            "causal_fraction_mean",
            "majority_excess_median",
            "majority_excess_mean",
            "majority_active_fraction",
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .median()
)

severity_order_628hc3a3a3a5 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

FD002_C3A3A3A5_FAMILY_SEVERITY[
    "_severity_order"
] = (
    FD002_C3A3A3A5_FAMILY_SEVERITY[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a5
    )
)

FD002_C3A3A3A5_FAMILY_SEVERITY = (
    FD002_C3A3A3A5_FAMILY_SEVERITY
    .sort_values(
        [
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(drop=True)
)

print(
    "\n--- FAMILY × SEVERITY CAUSAL-PERSISTENCE RESPONSE ---"
)

display(
    FD002_C3A3A3A5_FAMILY_SEVERITY
)

# ---------------------------------------------------------------
# 8. Strongest persistence scenarios
# ---------------------------------------------------------------

best_persistence_column_628hc3a3a3a5 = str(
    FD002_C3A3A3A5_UTILITY_ASSOCIATION
    .iloc[0][
        "persistence_summary"
    ]
)

FD002_C3A3A3A5_TOP_PERSISTENCE = (
    scenario_persistence_628hc3a3a3a5
    .sort_values(
        [
            best_persistence_column_628hc3a3a3a5,
            "delta_gain80_vs_uncertainty",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .head(25)
    [
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            best_persistence_column_628hc3a3a3a5,
            "delta_gain80_vs_uncertainty",
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\n--- TOP 25 SCENARIOS BY BEST "
    "CAUSAL-PERSISTENCE SUMMARY ---"
)

print(
    "Diagnostic display summary:",
    best_persistence_column_628hc3a3a3a5,
)

display(
    FD002_C3A3A3A5_TOP_PERSISTENCE
)

# ---------------------------------------------------------------
# 9. Integrity audit
# ---------------------------------------------------------------

all_scores_valid_628hc3a3a3a5 = bool(
    np.isfinite(
        causal_fraction_628hc3a3a3a5
    ).all()
    and
    np.isfinite(
        majority_excess_628hc3a3a3a5
    ).all()
    and
    np.isfinite(
        clean_causal_fraction_628hc3a3a3a5
    ).all()
    and
    np.isfinite(
        clean_majority_excess_628hc3a3a3a5
    ).all()
    and
    np.all(
        (
            causal_fraction_628hc3a3a3a5
            >=
            0.0
        )
        &
        (
            causal_fraction_628hc3a3a3a5
            <=
            1.0
        )
    )
    and
    np.all(
        (
            majority_excess_628hc3a3a3a5
            >=
            0.0
        )
        &
        (
            majority_excess_628hc3a3a3a5
            <=
            1.0
        )
    )
)

audit_628hc3a3a3a5 = pd.DataFrame(
    {
        "check": [
            "corrupted_shapes_exact",
            "clean_shapes_exact",
            "dev_window_keys_8367",
            "dev_window_keys_unique",
            "manifest_rows_315",
            "utility_rows_315",
            "scenario_summary_rows_315",
            "dev_engine_count_52",
            "five_association_rows",
            "family_severity_groups_15",
            "all_persistence_scores_valid",
        ],

        "passed": [

            all(
                arr_628hc3a3a3a5.shape
                ==
                EXPECTED_SHAPE_628HC3A3A3A5
                for arr_628hc3a3a3a5
                in (
                    uncertainty_628hc3a3a3a5,
                    noise_628hc3a3a3a5,
                    stuck_628hc3a3a3a5,
                )
            ),

            all(
                arr_628hc3a3a3a5.shape
                ==
                (8367,)
                for arr_628hc3a3a3a5
                in (
                    clean_uncertainty_628hc3a3a3a5,
                    clean_noise_628hc3a3a3a5,
                    clean_stuck_628hc3a3a3a5,
                )
            ),

            len(
                dev_keys_628hc3a3a3a5
            )
            ==
            8367,

            not (
                dev_keys_628hc3a3a3a5[
                    [
                        "unit_id",
                        "cycle",
                    ]
                ]
                .duplicated()
                .any()
            ),

            len(
                manifest_628hc3a3a3a5
            )
            ==
            315,

            len(
                utility_628hc3a3a3a5
            )
            ==
            315,

            len(
                FD002_C3A3A3A5_SCENARIO_PERSISTENCE
            )
            ==
            315,

            len(
                unique_units_628hc3a3a3a5
            )
            ==
            52,

            len(
                FD002_C3A3A3A5_UTILITY_ASSOCIATION
            )
            ==
            5,

            len(
                FD002_C3A3A3A5_FAMILY_SEVERITY
            )
            ==
            15,

            all_scores_valid_628hc3a3a3a5,
        ],
    }
)

print(
    "\n--- C3-A3-A3-A5 AUDIT ---"
)

display(
    audit_628hc3a3a3a5
)

all_checks_passed_628hc3a3a3a5 = bool(
    audit_628hc3a3a3a5[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3a5:
    raise RuntimeError(
        "C3-A3-A3-A5 causal-persistence audit failed."
    )

# ---------------------------------------------------------------
# 10. Final summary
# ---------------------------------------------------------------

best_row_628hc3a3a3a5 = (
    FD002_C3A3A3A5_UTILITY_ASSOCIATION
    .iloc[0]
)

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A5 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Evidence definition                  : "
    "noise_stuck_gap > 0"
)

print(
    "Persistence definition               : "
    "causal within-engine positive fraction"
)

print(
    "Pre-specified majority level          :",
    MAJORITY_LEVEL_628HC3A3A3A5,
)

print(
    "Best diagnostic persistence summary   :",
    best_row_628hc3a3a3a5[
        "persistence_summary"
    ],
)

print(
    "Best utility Spearman                 :",
    float(
        best_row_628hc3a3a3a5[
            "spearman_with_delta_gain80"
        ]
    ),
)

print(
    "All checks passed                     :",
    all_checks_passed_628hc3a3a3a5,
)

print("DEV RUL labels used                   : True")
print("Official TEST labels read             : False")
print("TEST scored                           : False")
print("Adaptive fusion rule selected         : False")
print("Combination weight selected           : False")
print("Data-tuned disagreement threshold     : False")
print("Abstention threshold selected         : False")
print("Model executed                        : False")
print("Model retrained                       : False")
print("Files written                         : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A5 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A6
# CAUSAL MAGNITUDE-DISAGREEMENT FUSION DIAGNOSTIC
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A6 — CAUSAL MAGNITUDE-DISAGREEMENT FUSION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Frozen aligned development objects
# ---------------------------------------------------------------

manifest_628hc3a3a3a6 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

dev_keys_628hc3a3a3a6 = (
    DEV_WINDOW_KEYS_FD002_628D
    .copy()
    .reset_index(drop=True)
)

prediction_628hc3a3a3a6 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64,
)

uncertainty_628hc3a3a3a6 = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

quality_628hc3a3a3a6 = np.asarray(
    FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C,
    dtype=np.float64,
)

noise_628hc3a3a3a6 = np.asarray(
    FD002_NOISE_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

stuck_628hc3a3a3a6 = np.asarray(
    FD002_STUCK_RISK_MATRIX_628HC3A3A2,
    dtype=np.float64,
)

y_dev_628hc3a3a3a6 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64,
).reshape(-1)

clean_prediction_628hc3a3a3a6 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
).reshape(-1)

clean_uncertainty_628hc3a3a3a6 = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_branch_dict_628hc3a3a3a6 = (
    CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B
)

clean_quality_628hc3a3a3a6 = np.asarray(
    clean_branch_dict_628hc3a3a3a6[
        "sensor_quality_risk"
    ],
    dtype=np.float64,
).reshape(-1)

clean_noise_628hc3a3a3a6 = np.asarray(
    clean_branch_dict_628hc3a3a3a6[
        "noise"
    ],
    dtype=np.float64,
).reshape(-1)

clean_stuck_628hc3a3a3a6 = np.asarray(
    clean_branch_dict_628hc3a3a3a6[
        "stuck"
    ],
    dtype=np.float64,
).reshape(-1)

EXPECTED_SHAPE_628HC3A3A3A6 = (
    315,
    8367,
)

for matrix_628hc3a3a3a6 in (
    prediction_628hc3a3a3a6,
    uncertainty_628hc3a3a3a6,
    quality_628hc3a3a3a6,
    noise_628hc3a3a3a6,
    stuck_628hc3a3a3a6,
):
    if (
        matrix_628hc3a3a3a6.shape
        !=
        EXPECTED_SHAPE_628HC3A3A3A6
    ):
        raise RuntimeError(
            "Unexpected corrupted matrix shape: "
            f"{matrix_628hc3a3a3a6.shape}"
        )

for vector_628hc3a3a3a6 in (
    y_dev_628hc3a3a3a6,
    clean_prediction_628hc3a3a3a6,
    clean_uncertainty_628hc3a3a3a6,
    clean_quality_628hc3a3a3a6,
    clean_noise_628hc3a3a3a6,
    clean_stuck_628hc3a3a3a6,
):
    if vector_628hc3a3a3a6.shape != (8367,):
        raise RuntimeError(
            "Unexpected clean vector shape: "
            f"{vector_628hc3a3a3a6.shape}"
        )

if len(dev_keys_628hc3a3a3a6) != 8367:
    raise RuntimeError(
        "DEV window-key count is not 8367."
    )

if (
    dev_keys_628hc3a3a3a6[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "Duplicate DEV window keys detected."
    )

if not np.array_equal(
    manifest_628hc3a3a3a6[
        "scenario_number"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Scenario order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 2. Fixed diagnostic contract
# ---------------------------------------------------------------

GAMMA_628HC3A3A3A6 = 0.80

COVERAGES_628HC3A3A3A6 = (
    0.90,
    0.80,
    0.70,
)

QUALITY_Q95_628HC3A3A3A6 = float(
    FIT_SENSOR_QUALITY_Q95_MARKER_628HC2B
)

# ---------------------------------------------------------------
# 3. Instantaneous magnitude disagreement
# ---------------------------------------------------------------

noise_stuck_628hc3a3a3a6 = np.minimum(
    noise_628hc3a3a3a6,
    stuck_628hc3a3a3a6,
)

clean_noise_stuck_628hc3a3a3a6 = np.minimum(
    clean_noise_628hc3a3a3a6,
    clean_stuck_628hc3a3a3a6,
)

instant_gap_628hc3a3a3a6 = np.clip(
    noise_stuck_628hc3a3a3a6
    -
    uncertainty_628hc3a3a3a6,
    0.0,
    1.0,
)

clean_instant_gap_628hc3a3a3a6 = np.clip(
    clean_noise_stuck_628hc3a3a3a6
    -
    clean_uncertainty_628hc3a3a3a6,
    0.0,
    1.0,
)

# ---------------------------------------------------------------
# 4. Causal within-engine mean disagreement magnitude
#
# Only current + earlier windows from the same engine are used.
# No future information is used.
# ---------------------------------------------------------------

causal_mean_gap_628hc3a3a3a6 = np.zeros_like(
    instant_gap_628hc3a3a3a6,
    dtype=np.float64,
)

clean_causal_mean_gap_628hc3a3a3a6 = np.zeros(
    8367,
    dtype=np.float64,
)

unit_ids_628hc3a3a3a6 = (
    dev_keys_628hc3a3a3a6[
        "unit_id"
    ]
    .to_numpy()
)

cycles_628hc3a3a3a6 = (
    dev_keys_628hc3a3a3a6[
        "cycle"
    ]
    .to_numpy()
)

unique_units_628hc3a3a3a6 = np.unique(
    unit_ids_628hc3a3a3a6
)

if len(unique_units_628hc3a3a3a6) != 52:
    raise RuntimeError(
        "Expected exactly 52 DEV engines."
    )

for unit_id_628hc3a3a3a6 in (
    unique_units_628hc3a3a3a6
):

    positions_628hc3a3a3a6 = np.flatnonzero(
        unit_ids_628hc3a3a3a6
        ==
        unit_id_628hc3a3a3a6
    )

    unit_cycles_628hc3a3a3a6 = (
        cycles_628hc3a3a3a6[
            positions_628hc3a3a3a6
        ]
    )

    order_628hc3a3a3a6 = np.argsort(
        unit_cycles_628hc3a3a3a6,
        kind="mergesort",
    )

    ordered_positions_628hc3a3a3a6 = (
        positions_628hc3a3a3a6[
            order_628hc3a3a3a6
        ]
    )

    n_windows_628hc3a3a3a6 = len(
        ordered_positions_628hc3a3a3a6
    )

    denominator_628hc3a3a3a6 = np.arange(
        1,
        n_windows_628hc3a3a3a6 + 1,
        dtype=np.float64,
    )

    corrupted_unit_gap_628hc3a3a3a6 = (
        instant_gap_628hc3a3a3a6[
            :,
            ordered_positions_628hc3a3a3a6,
        ]
    )

    causal_mean_gap_628hc3a3a3a6[
        :,
        ordered_positions_628hc3a3a3a6,
    ] = (
        np.cumsum(
            corrupted_unit_gap_628hc3a3a3a6,
            axis=1,
        )
        /
        denominator_628hc3a3a3a6[
            None,
            :
        ]
    )

    clean_unit_gap_628hc3a3a3a6 = (
        clean_instant_gap_628hc3a3a3a6[
            ordered_positions_628hc3a3a3a6
        ]
    )

    clean_causal_mean_gap_628hc3a3a3a6[
        ordered_positions_628hc3a3a3a6
    ] = (
        np.cumsum(
            clean_unit_gap_628hc3a3a3a6
        )
        /
        denominator_628hc3a3a3a6
    )

# ---------------------------------------------------------------
# 5. Diagnostic gate baseline
# ---------------------------------------------------------------

gate_baseline_rows_628hc3a3a3a6 = [
    {
        "gate":
            "instantaneous_gap",

        "clean_median":
            float(
                np.median(
                    clean_instant_gap_628hc3a3a3a6
                )
            ),

        "clean_mean":
            float(
                np.mean(
                    clean_instant_gap_628hc3a3a3a6
                )
            ),

        "clean_q90":
            float(
                np.quantile(
                    clean_instant_gap_628hc3a3a3a6,
                    0.90,
                )
            ),

        "clean_q95":
            float(
                np.quantile(
                    clean_instant_gap_628hc3a3a3a6,
                    0.95,
                )
            ),
    },

    {
        "gate":
            "causal_mean_gap",

        "clean_median":
            float(
                np.median(
                    clean_causal_mean_gap_628hc3a3a3a6
                )
            ),

        "clean_mean":
            float(
                np.mean(
                    clean_causal_mean_gap_628hc3a3a3a6
                )
            ),

        "clean_q90":
            float(
                np.quantile(
                    clean_causal_mean_gap_628hc3a3a3a6,
                    0.90,
                )
            ),

        "clean_q95":
            float(
                np.quantile(
                    clean_causal_mean_gap_628hc3a3a3a6,
                    0.95,
                )
            ),
    },
]

FD002_C3A3A3A6_GATE_BASELINE = (
    pd.DataFrame(
        gate_baseline_rows_628hc3a3a3a6
    )
)

print(
    "\n--- CLEAN DEV MAGNITUDE-GATE BASELINE ---"
)

display(
    FD002_C3A3A3A6_GATE_BASELINE
)

# ---------------------------------------------------------------
# 6. Candidate set
# ---------------------------------------------------------------

CANDIDATES_628HC3A3A3A6 = (
    "uncertainty_only",
    "fixed_global_g080",
    "instantaneous_gap_g080",
    "causal_mean_gap_g080",
)

print(
    "\n--- CANDIDATES ---"
)

for candidate_628hc3a3a3a6 in (
    CANDIDATES_628HC3A3A3A6
):
    print(
        candidate_628hc3a3a3a6
    )

# ---------------------------------------------------------------
# 7. Helpers
# ---------------------------------------------------------------

def quality_excess_628hc3a3a3a6(
    quality_values
):
    quality_values = np.asarray(
        quality_values,
        dtype=np.float64,
    )

    denominator = (
        1.0
        -
        QUALITY_Q95_628HC3A3A3A6
    )

    if denominator <= 0.0:
        raise RuntimeError(
            "Invalid FIT quality q95 marker."
        )

    return np.clip(
        (
            quality_values
            -
            QUALITY_Q95_628HC3A3A3A6
        )
        /
        denominator,
        0.0,
        1.0,
    )


def fused_risk_628hc3a3a3a6(
    uncertainty_values,
    quality_values,
    gate_values,
):
    uncertainty_values = np.asarray(
        uncertainty_values,
        dtype=np.float64,
    )

    quality_values = np.asarray(
        quality_values,
        dtype=np.float64,
    )

    gate_values = np.asarray(
        gate_values,
        dtype=np.float64,
    )

    if not (
        uncertainty_values.shape
        ==
        quality_values.shape
        ==
        gate_values.shape
    ):
        raise ValueError(
            "U, Q, and gate arrays must have identical shapes."
        )

    excess = (
        quality_excess_628hc3a3a3a6(
            quality_values
        )
    )

    risk = (
        uncertainty_values
        +
        (
            1.0
            -
            uncertainty_values
        )
        *
        GAMMA_628HC3A3A3A6
        *
        gate_values
        *
        excess
    )

    return np.clip(
        risk,
        0.0,
        1.0,
    )


def make_candidate_risk_628hc3a3a3a6(
    candidate,
    uncertainty_values,
    quality_values,
    instant_gate,
    causal_gate,
):
    if candidate == "uncertainty_only":
        return np.asarray(
            uncertainty_values,
            dtype=np.float64,
        ).copy()

    if candidate == "fixed_global_g080":
        gate_values = np.ones_like(
            uncertainty_values,
            dtype=np.float64,
        )

    elif candidate == "instantaneous_gap_g080":
        gate_values = instant_gate

    elif candidate == "causal_mean_gap_g080":
        gate_values = causal_gate

    else:
        raise ValueError(
            f"Unknown candidate: {candidate}"
        )

    return fused_risk_628hc3a3a3a6(
        uncertainty_values,
        quality_values,
        gate_values,
    )


def evaluate_coverages_628hc3a3a3a6(
    risk_values,
    absolute_error,
):
    full_mae = float(
        np.mean(
            absolute_error
        )
    )

    result = {
        "full_mae":
            full_mae,
    }

    for coverage_628hc3a3a3a6 in (
        COVERAGES_628HC3A3A3A6
    ):

        selective_result_628hc3a3a3a6 = (
            selective_mae_tie_aware_628hc3a2(
                risk_values,
                absolute_error,
                coverage_628hc3a3a3a6,
            )
        )

        pct_628hc3a3a3a6 = int(
            round(
                coverage_628hc3a3a3a6
                *
                100
            )
        )

        mae_628hc3a3a3a6 = float(
            selective_result_628hc3a3a3a6[
                "mae"
            ]
        )

        result[
            f"mae_{pct_628hc3a3a3a6}"
        ] = mae_628hc3a3a3a6

        result[
            f"gain_{pct_628hc3a3a3a6}"
        ] = float(
            full_mae
            -
            mae_628hc3a3a3a6
        )

    return result

# ---------------------------------------------------------------
# 8. Clean DEV evaluation
# ---------------------------------------------------------------

clean_error_628hc3a3a3a6 = np.abs(
    clean_prediction_628hc3a3a3a6
    -
    y_dev_628hc3a3a3a6
)

clean_rows_628hc3a3a3a6 = []

for candidate_628hc3a3a3a6 in (
    CANDIDATES_628HC3A3A3A6
):

    clean_risk_628hc3a3a3a6 = (
        make_candidate_risk_628hc3a3a3a6(
            candidate_628hc3a3a3a6,
            clean_uncertainty_628hc3a3a3a6,
            clean_quality_628hc3a3a3a6,
            clean_instant_gap_628hc3a3a3a6,
            clean_causal_mean_gap_628hc3a3a3a6,
        )
    )

    metrics_628hc3a3a3a6 = (
        evaluate_coverages_628hc3a3a3a6(
            clean_risk_628hc3a3a3a6,
            clean_error_628hc3a3a3a6,
        )
    )

    clean_rows_628hc3a3a3a6.append(
        {
            "candidate":
                candidate_628hc3a3a3a6,

            **metrics_628hc3a3a3a6,
        }
    )

FD002_C3A3A3A6_CLEAN_RESPONSE = (
    pd.DataFrame(
        clean_rows_628hc3a3a3a6
    )
)

print(
    "\n--- CLEAN DEV CANDIDATE RESPONSE ---"
)

display(
    FD002_C3A3A3A6_CLEAN_RESPONSE
)

# ---------------------------------------------------------------
# 9. Full scenario evaluation
# ---------------------------------------------------------------

scenario_rows_628hc3a3a3a6 = []

for candidate_628hc3a3a3a6 in (
    CANDIDATES_628HC3A3A3A6
):

    for i_628hc3a3a3a6 in range(
        315
    ):

        risk_628hc3a3a3a6 = (
            make_candidate_risk_628hc3a3a3a6(
                candidate_628hc3a3a3a6,
                uncertainty_628hc3a3a3a6[
                    i_628hc3a3a3a6
                ],
                quality_628hc3a3a3a6[
                    i_628hc3a3a3a6
                ],
                instant_gap_628hc3a3a3a6[
                    i_628hc3a3a3a6
                ],
                causal_mean_gap_628hc3a3a3a6[
                    i_628hc3a3a3a6
                ],
            )
        )

        error_628hc3a3a3a6 = np.abs(
            prediction_628hc3a3a3a6[
                i_628hc3a3a3a6
            ]
            -
            y_dev_628hc3a3a3a6
        )

        metrics_628hc3a3a3a6 = (
            evaluate_coverages_628hc3a3a3a6(
                risk_628hc3a3a3a6,
                error_628hc3a3a3a6,
            )
        )

        manifest_row_628hc3a3a3a6 = (
            manifest_628hc3a3a3a6.iloc[
                i_628hc3a3a3a6
            ]
        )

        scenario_rows_628hc3a3a3a6.append(
            {
                "candidate":
                    candidate_628hc3a3a3a6,

                "scenario_id":
                    str(
                        manifest_row_628hc3a3a3a6[
                            "scenario_id"
                        ]
                    ),

                "scenario_number":
                    int(
                        manifest_row_628hc3a3a3a6[
                            "scenario_number"
                        ]
                    ),

                "family":
                    str(
                        manifest_row_628hc3a3a3a6[
                            "family"
                        ]
                    ),

                "sensor":
                    str(
                        manifest_row_628hc3a3a3a6[
                            "sensor"
                        ]
                    ),

                "severity":
                    str(
                        manifest_row_628hc3a3a3a6[
                            "severity"
                        ]
                    ),

                **metrics_628hc3a3a3a6,
            }
        )

FD002_C3A3A3A6_SCENARIO_METRICS = (
    pd.DataFrame(
        scenario_rows_628hc3a3a3a6
    )
)

expected_rows_628hc3a3a3a6 = (
    315
    *
    len(
        CANDIDATES_628HC3A3A3A6
    )
)

if (
    len(
        FD002_C3A3A3A6_SCENARIO_METRICS
    )
    !=
    expected_rows_628hc3a3a3a6
):
    raise RuntimeError(
        "Unexpected candidate-scenario row count."
    )

# ---------------------------------------------------------------
# 10. Overall summary
# ---------------------------------------------------------------

baseline_628hc3a3a3a6 = (
    FD002_C3A3A3A6_SCENARIO_METRICS[
        FD002_C3A3A3A6_SCENARIO_METRICS[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ]
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

summary_rows_628hc3a3a3a6 = []

for candidate_628hc3a3a3a6 in (
    CANDIDATES_628HC3A3A3A6
):

    candidate_df_628hc3a3a3a6 = (
        FD002_C3A3A3A6_SCENARIO_METRICS[
            FD002_C3A3A3A6_SCENARIO_METRICS[
                "candidate"
            ]
            ==
            candidate_628hc3a3a3a6
        ]
        .sort_values(
            "scenario_number",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    result_row_628hc3a3a3a6 = {
        "candidate":
            candidate_628hc3a3a3a6,
    }

    for pct_628hc3a3a3a6 in (
        90,
        80,
        70,
    ):

        gain_col_628hc3a3a3a6 = (
            f"gain_{pct_628hc3a3a3a6}"
        )

        gains_628hc3a3a3a6 = (
            candidate_df_628hc3a3a3a6[
                gain_col_628hc3a3a3a6
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        baseline_gains_628hc3a3a3a6 = (
            baseline_628hc3a3a3a6[
                gain_col_628hc3a3a3a6
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        delta_628hc3a3a3a6 = (
            gains_628hc3a3a3a6
            -
            baseline_gains_628hc3a3a3a6
        )

        family_medians_628hc3a3a3a6 = (
            candidate_df_628hc3a3a3a6
            .groupby(
                "family",
                observed=True,
            )[
                gain_col_628hc3a3a3a6
            ]
            .median()
        )

        result_row_628hc3a3a3a6[
            f"median_gain_{pct_628hc3a3a3a6}"
        ] = float(
            np.median(
                gains_628hc3a3a3a6
            )
        )

        result_row_628hc3a3a3a6[
            f"mean_gain_{pct_628hc3a3a3a6}"
        ] = float(
            np.mean(
                gains_628hc3a3a3a6
            )
        )

        result_row_628hc3a3a3a6[
            f"q10_gain_{pct_628hc3a3a3a6}"
        ] = float(
            np.quantile(
                gains_628hc3a3a3a6,
                0.10,
            )
        )

        result_row_628hc3a3a3a6[
            f"worst_family_median_gain_{pct_628hc3a3a3a6}"
        ] = float(
            family_medians_628hc3a3a3a6.min()
        )

        result_row_628hc3a3a3a6[
            f"median_delta_vs_u_{pct_628hc3a3a3a6}"
        ] = float(
            np.median(
                delta_628hc3a3a3a6
            )
        )

        result_row_628hc3a3a3a6[
            f"mean_delta_vs_u_{pct_628hc3a3a3a6}"
        ] = float(
            np.mean(
                delta_628hc3a3a3a6
            )
        )

        result_row_628hc3a3a3a6[
            f"improved_vs_u_{pct_628hc3a3a3a6}"
        ] = int(
            np.sum(
                delta_628hc3a3a3a6
                >
                1e-12
            )
        )

        result_row_628hc3a3a3a6[
            f"equal_vs_u_{pct_628hc3a3a3a6}"
        ] = int(
            np.sum(
                np.abs(
                    delta_628hc3a3a3a6
                )
                <=
                1e-12
            )
        )

        result_row_628hc3a3a3a6[
            f"worse_vs_u_{pct_628hc3a3a3a6}"
        ] = int(
            np.sum(
                delta_628hc3a3a3a6
                <
                -1e-12
            )
        )

    summary_rows_628hc3a3a3a6.append(
        result_row_628hc3a3a3a6
    )

FD002_C3A3A3A6_ARCHITECTURE_SUMMARY = (
    pd.DataFrame(
        summary_rows_628hc3a3a3a6
    )
)

print(
    "\n--- CAUSAL MAGNITUDE ARCHITECTURE SUMMARY ---"
)

display(
    FD002_C3A3A3A6_ARCHITECTURE_SUMMARY
)

# ---------------------------------------------------------------
# 11. Family × severity @80%
# ---------------------------------------------------------------

FD002_C3A3A3A6_FAMILY_SEVERITY = (
    FD002_C3A3A3A6_SCENARIO_METRICS
    .groupby(
        [
            "candidate",
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        n_scenarios=(
            "scenario_number",
            "size",
        ),

        median_gain80=(
            "gain_80",
            "median",
        ),

        mean_gain80=(
            "gain_80",
            "mean",
        ),
    )
)

severity_order_628hc3a3a3a6 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

FD002_C3A3A3A6_FAMILY_SEVERITY[
    "_severity_order"
] = (
    FD002_C3A3A3A6_FAMILY_SEVERITY[
        "severity"
    ]
    .map(
        severity_order_628hc3a3a3a6
    )
)

FD002_C3A3A3A6_FAMILY_SEVERITY = (
    FD002_C3A3A3A6_FAMILY_SEVERITY
    .sort_values(
        [
            "candidate",
            "family",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(drop=True)
)

FD002_C3A3A3A6_STUCK_RESPONSE = (
    FD002_C3A3A3A6_FAMILY_SEVERITY[
        FD002_C3A3A3A6_FAMILY_SEVERITY[
            "family"
        ]
        ==
        "stuck"
    ]
    .reset_index(drop=True)
)

print(
    "\n--- STUCK-FAMILY RESPONSE @80% COVERAGE ---"
)

display(
    FD002_C3A3A3A6_STUCK_RESPONSE
)

# ---------------------------------------------------------------
# 12. Clean compact response
# ---------------------------------------------------------------

print(
    "\n--- CLEAN DEV SELECTIVE GAINS ---"
)

display(
    FD002_C3A3A3A6_CLEAN_RESPONSE[
        [
            "candidate",
            "gain_90",
            "gain_80",
            "gain_70",
        ]
    ]
)

# ---------------------------------------------------------------
# 13. Integrity audit
# ---------------------------------------------------------------

metric_columns_628hc3a3a3a6 = [
    "full_mae",
    "mae_90",
    "gain_90",
    "mae_80",
    "gain_80",
    "mae_70",
    "gain_70",
]

all_gate_values_valid_628hc3a3a3a6 = bool(
    np.isfinite(
        instant_gap_628hc3a3a3a6
    ).all()
    and
    np.isfinite(
        causal_mean_gap_628hc3a3a3a6
    ).all()
    and
    np.isfinite(
        clean_instant_gap_628hc3a3a3a6
    ).all()
    and
    np.isfinite(
        clean_causal_mean_gap_628hc3a3a3a6
    ).all()
    and
    np.all(
        (
            causal_mean_gap_628hc3a3a3a6
            >=
            0.0
        )
        &
        (
            causal_mean_gap_628hc3a3a3a6
            <=
            1.0
        )
    )
)

audit_628hc3a3a3a6 = pd.DataFrame(
    {
        "check": [
            "candidate_count_4",
            "scenario_count_315",
            "dev_windows_8367",
            "dev_engines_52",
            "scenario_candidate_rows_exact",
            "clean_candidate_rows_4",
            "three_coverages_evaluated",
            "all_metrics_finite",
            "all_gate_values_valid",
            "scenario_order_exact",
        ],

        "passed": [

            len(
                CANDIDATES_628HC3A3A3A6
            )
            ==
            4,

            len(
                manifest_628hc3a3a3a6
            )
            ==
            315,

            len(
                dev_keys_628hc3a3a3a6
            )
            ==
            8367,

            len(
                unique_units_628hc3a3a3a6
            )
            ==
            52,

            len(
                FD002_C3A3A3A6_SCENARIO_METRICS
            )
            ==
            expected_rows_628hc3a3a3a6,

            len(
                FD002_C3A3A3A6_CLEAN_RESPONSE
            )
            ==
            4,

            len(
                COVERAGES_628HC3A3A3A6
            )
            ==
            3,

            bool(
                np.isfinite(
                    FD002_C3A3A3A6_SCENARIO_METRICS[
                        metric_columns_628hc3a3a3a6
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                )
                .all()
            ),

            all_gate_values_valid_628hc3a3a3a6,

            bool(
                np.array_equal(
                    manifest_628hc3a3a3a6[
                        "scenario_number"
                    ]
                    .to_numpy(
                        dtype=np.int64
                    ),
                    np.arange(
                        1,
                        316,
                        dtype=np.int64,
                    ),
                )
            ),
        ],
    }
)

print(
    "\n--- C3-A3-A3-A6 AUDIT ---"
)

display(
    audit_628hc3a3a3a6
)

all_checks_passed_628hc3a3a3a6 = bool(
    audit_628hc3a3a3a6[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3a6:
    raise RuntimeError(
        "C3-A3-A3-A6 audit failed."
    )

# ---------------------------------------------------------------
# 14. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A6 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Architectures evaluated             :",
    len(
        CANDIDATES_628HC3A3A3A6
    ),
)

print(
    "Diagnostic gamma                    :",
    GAMMA_628HC3A3A3A6,
)

print(
    "New gate threshold selected         :",
    False,
)

print(
    "Causal future leakage               :",
    False,
)

print(
    "All checks passed                   :",
    all_checks_passed_628hc3a3a3a6,
)

print("DEV RUL labels used                 : True")
print("Official TEST labels read           : False")
print("TEST scored                         : False")
print("Adaptive fusion rule selected       : False")
print("Combination weight selected         : False")
print("Abstention threshold selected       : False")
print("Model executed                      : False")
print("Model retrained                     : False")
print("Files written                       : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A6 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-A7
# ADAPTIVE-FUSION STOPPING / DECISION AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-A7 — ADAPTIVE-FUSION STOPPING / DECISION AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Use only already-computed A6 results
# ---------------------------------------------------------------

clean_628hc3a3a3a7 = (
    FD002_C3A3A3A6_CLEAN_RESPONSE
    .copy()
    .set_index("candidate")
)

stuck_628hc3a3a3a7 = (
    FD002_C3A3A3A6_STUCK_RESPONSE
    .copy()
)

required_candidates_628hc3a3a3a7 = [
    "uncertainty_only",
    "fixed_global_g080",
    "instantaneous_gap_g080",
    "causal_mean_gap_g080",
]

missing_candidates_628hc3a3a3a7 = (
    set(required_candidates_628hc3a3a3a7)
    -
    set(clean_628hc3a3a3a7.index)
)

if missing_candidates_628hc3a3a3a7:
    raise RuntimeError(
        "Missing A6 candidates: "
        f"{sorted(missing_candidates_628hc3a3a3a7)}"
    )

if len(stuck_628hc3a3a3a7) != 12:
    raise RuntimeError(
        "Expected exactly 12 stuck-family rows."
    )

# ---------------------------------------------------------------
# 2. Uncertainty-only reference
# ---------------------------------------------------------------

baseline_clean_628hc3a3a3a7 = {
    90: float(
        clean_628hc3a3a3a7.loc[
            "uncertainty_only",
            "gain_90",
        ]
    ),

    80: float(
        clean_628hc3a3a3a7.loc[
            "uncertainty_only",
            "gain_80",
        ]
    ),

    70: float(
        clean_628hc3a3a3a7.loc[
            "uncertainty_only",
            "gain_70",
        ]
    ),
}

baseline_stuck_628hc3a3a3a7 = (
    stuck_628hc3a3a3a7[
        stuck_628hc3a3a3a7[
            "candidate"
        ]
        ==
        "uncertainty_only"
    ]
    .set_index(
        "severity"
    )[
        "median_gain80"
    ]
)

if set(
    baseline_stuck_628hc3a3a3a7.index
) != {
    "low",
    "medium",
    "high",
}:
    raise RuntimeError(
        "Uncertainty-only stuck response is incomplete."
    )

# ---------------------------------------------------------------
# 3. Acceptance criteria
#
# These formalize the design goals already established before A6.
#
# An adaptive candidate must:
#
# A. preserve clean selective gain at 90%, 80%, and 70%;
# B. not worsen low-stuck @80% versus uncertainty-only;
# C. produce positive selective gain for medium-stuck @80%;
# D. produce positive selective gain for high-stuck @80%.
#
# No new architecture or tuning occurs here.
# ---------------------------------------------------------------

TOL_628HC3A3A3A7 = 1e-12

adaptive_candidates_628hc3a3a3a7 = [
    "fixed_global_g080",
    "instantaneous_gap_g080",
    "causal_mean_gap_g080",
]

decision_rows_628hc3a3a3a7 = []

for candidate_628hc3a3a3a7 in (
    adaptive_candidates_628hc3a3a3a7
):

    clean_gain90_628hc3a3a3a7 = float(
        clean_628hc3a3a3a7.loc[
            candidate_628hc3a3a3a7,
            "gain_90",
        ]
    )

    clean_gain80_628hc3a3a3a7 = float(
        clean_628hc3a3a3a7.loc[
            candidate_628hc3a3a3a7,
            "gain_80",
        ]
    )

    clean_gain70_628hc3a3a3a7 = float(
        clean_628hc3a3a3a7.loc[
            candidate_628hc3a3a3a7,
            "gain_70",
        ]
    )

    stuck_candidate_628hc3a3a3a7 = (
        stuck_628hc3a3a3a7[
            stuck_628hc3a3a3a7[
                "candidate"
            ]
            ==
            candidate_628hc3a3a3a7
        ]
        .set_index(
            "severity"
        )[
            "median_gain80"
        ]
    )

    if set(
        stuck_candidate_628hc3a3a3a7.index
    ) != {
        "low",
        "medium",
        "high",
    }:
        raise RuntimeError(
            "Incomplete stuck response for "
            f"{candidate_628hc3a3a3a7}."
        )

    stuck_low_628hc3a3a3a7 = float(
        stuck_candidate_628hc3a3a3a7[
            "low"
        ]
    )

    stuck_medium_628hc3a3a3a7 = float(
        stuck_candidate_628hc3a3a3a7[
            "medium"
        ]
    )

    stuck_high_628hc3a3a3a7 = float(
        stuck_candidate_628hc3a3a3a7[
            "high"
        ]
    )

    preserves_clean_628hc3a3a3a7 = bool(
        clean_gain90_628hc3a3a3a7
        >=
        baseline_clean_628hc3a3a3a7[90]
        -
        TOL_628HC3A3A3A7

        and

        clean_gain80_628hc3a3a3a7
        >=
        baseline_clean_628hc3a3a3a7[80]
        -
        TOL_628HC3A3A3A7

        and

        clean_gain70_628hc3a3a3a7
        >=
        baseline_clean_628hc3a3a3a7[70]
        -
        TOL_628HC3A3A3A7
    )

    preserves_stuck_low_628hc3a3a3a7 = bool(
        stuck_low_628hc3a3a3a7
        >=
        float(
            baseline_stuck_628hc3a3a3a7[
                "low"
            ]
        )
        -
        TOL_628HC3A3A3A7
    )

    repairs_stuck_medium_628hc3a3a3a7 = bool(
        stuck_medium_628hc3a3a3a7
        >
        0.0
    )

    repairs_stuck_high_628hc3a3a3a7 = bool(
        stuck_high_628hc3a3a3a7
        >
        0.0
    )

    acceptance_passed_628hc3a3a3a7 = bool(
        preserves_clean_628hc3a3a3a7
        and
        preserves_stuck_low_628hc3a3a3a7
        and
        repairs_stuck_medium_628hc3a3a3a7
        and
        repairs_stuck_high_628hc3a3a3a7
    )

    decision_rows_628hc3a3a3a7.append(
        {
            "candidate":
                candidate_628hc3a3a3a7,

            "clean_gain90":
                clean_gain90_628hc3a3a3a7,

            "clean_gain80":
                clean_gain80_628hc3a3a3a7,

            "clean_gain70":
                clean_gain70_628hc3a3a3a7,

            "stuck_low_gain80":
                stuck_low_628hc3a3a3a7,

            "stuck_medium_gain80":
                stuck_medium_628hc3a3a3a7,

            "stuck_high_gain80":
                stuck_high_628hc3a3a3a7,

            "preserves_clean_90_80_70":
                preserves_clean_628hc3a3a3a7,

            "preserves_stuck_low":
                preserves_stuck_low_628hc3a3a3a7,

            "repairs_stuck_medium_positive":
                repairs_stuck_medium_628hc3a3a3a7,

            "repairs_stuck_high_positive":
                repairs_stuck_high_628hc3a3a3a7,

            "adaptive_acceptance_passed":
                acceptance_passed_628hc3a3a3a7,
        }
    )

FD002_C3A3A3A7_ADAPTIVE_DECISION = (
    pd.DataFrame(
        decision_rows_628hc3a3a3a7
    )
)

print(
    "\n--- ADAPTIVE-FUSION ACCEPTANCE AUDIT ---"
)

display(
    FD002_C3A3A3A7_ADAPTIVE_DECISION
)

# ---------------------------------------------------------------
# 4. Decision closure
# ---------------------------------------------------------------

passing_candidates_628hc3a3a3a7 = (
    FD002_C3A3A3A7_ADAPTIVE_DECISION.loc[
        FD002_C3A3A3A7_ADAPTIVE_DECISION[
            "adaptive_acceptance_passed"
        ],
        "candidate",
    ]
    .tolist()
)

if len(
    passing_candidates_628hc3a3a3a7
) == 0:

    FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED = (
        False
    )

    FD002_C3A3A3A7_SELECTED_ARCHITECTURE = (
        "uncertainty_only"
    )

    FD002_C3A3A3A7_DECISION = (
        "retain_uncertainty_only"
    )

else:

    raise RuntimeError(
        "An adaptive candidate passed every criterion. "
        "Do not auto-select it here; perform a separate "
        "robustness comparison."
    )

# ---------------------------------------------------------------
# 5. Integrity audit
# ---------------------------------------------------------------

audit_628hc3a3a3a7 = pd.DataFrame(
    {
        "check": [
            "all_required_A6_candidates_present",
            "stuck_rows_12",
            "three_adaptive_candidates_checked",
            "no_new_architecture",
            "no_new_gamma",
            "no_new_gate_threshold",
            "adaptive_pass_count_zero",
            "fallback_is_uncertainty_only",
        ],

        "passed": [
            len(
                missing_candidates_628hc3a3a3a7
            )
            ==
            0,

            len(
                stuck_628hc3a3a3a7
            )
            ==
            12,

            len(
                FD002_C3A3A3A7_ADAPTIVE_DECISION
            )
            ==
            3,

            True,

            True,

            True,

            len(
                passing_candidates_628hc3a3a3a7
            )
            ==
            0,

            FD002_C3A3A3A7_SELECTED_ARCHITECTURE
            ==
            "uncertainty_only",
        ],
    }
)

print(
    "\n--- C3-A3-A3-A7 AUDIT ---"
)

display(
    audit_628hc3a3a3a7
)

all_checks_passed_628hc3a3a3a7 = bool(
    audit_628hc3a3a3a7[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3a7:
    raise RuntimeError(
        "C3-A3-A3-A7 decision audit failed."
    )

# ---------------------------------------------------------------
# 6. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-A7 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Adaptive candidates evaluated        :",
    len(
        adaptive_candidates_628hc3a3a3a7
    ),
)

print(
    "Adaptive candidates passing all goals:",
    len(
        passing_candidates_628hc3a3a3a7
    ),
)

print(
    "Adaptive fusion accepted             :",
    FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED,
)

print(
    "Selected reliability architecture    :",
    FD002_C3A3A3A7_SELECTED_ARCHITECTURE,
)

print(
    "Decision                             :",
    FD002_C3A3A3A7_DECISION,
)

print("DEV RUL labels used                  : True")
print("Official TEST labels read            : False")
print("TEST scored                          : False")
print("New architecture introduced          : False")
print("New gamma selected                   : False")
print("New gate threshold selected          : False")
print("Abstention threshold selected        : False")
print("Model executed                       : False")
print("Model retrained                      : False")
print("Files written                        : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-A7 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-F2-A
# FREEZE FD002 SELECTED RELIABILITY ARCHITECTURE
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-F2-A — FREEZE FD002 SELECTED RELIABILITY ARCHITECTURE")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Lock the A7 development decision
# ---------------------------------------------------------------

selected_architecture_628hc3a3a3f2a = (
    FD002_C3A3A3A7_SELECTED_ARCHITECTURE
)

decision_628hc3a3a3f2a = (
    FD002_C3A3A3A7_DECISION
)

adaptive_accepted_628hc3a3a3f2a = bool(
    FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED
)

decision_table_628hc3a3a3f2a = (
    FD002_C3A3A3A7_ADAPTIVE_DECISION
    .copy()
    .reset_index(drop=True)
)

if selected_architecture_628hc3a3a3f2a != "uncertainty_only":
    raise RuntimeError(
        "A7 selected architecture is not uncertainty_only."
    )

if decision_628hc3a3a3f2a != "retain_uncertainty_only":
    raise RuntimeError(
        "A7 decision is not retain_uncertainty_only."
    )

if adaptive_accepted_628hc3a3a3f2a:
    raise RuntimeError(
        "Adaptive fusion was unexpectedly marked accepted."
    )

if bool(
    decision_table_628hc3a3a3f2a[
        "adaptive_acceptance_passed"
    ].any()
):
    raise RuntimeError(
        "At least one adaptive candidate is marked as accepted."
    )

# ---------------------------------------------------------------
# 2. Recover the exact selected reliability-risk objects
# ---------------------------------------------------------------

corrupted_selected_risk_628hc3a3a3f2a = np.asarray(
    FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
    dtype=np.float64,
)

clean_selected_risk_628hc3a3a3f2a = np.asarray(
    FD002_DEV_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

manifest_628hc3a3a3f2a = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_corrupted_shape_628hc3a3a3f2a = (
    315,
    8367,
)

if (
    corrupted_selected_risk_628hc3a3a3f2a.shape
    !=
    expected_corrupted_shape_628hc3a3a3f2a
):
    raise RuntimeError(
        "Unexpected corrupted selected-risk shape: "
        f"{corrupted_selected_risk_628hc3a3a3f2a.shape}"
    )

if clean_selected_risk_628hc3a3a3f2a.shape != (8367,):
    raise RuntimeError(
        "Unexpected clean selected-risk shape: "
        f"{clean_selected_risk_628hc3a3a3f2a.shape}"
    )

for risk_628hc3a3a3f2a in (
    corrupted_selected_risk_628hc3a3a3f2a,
    clean_selected_risk_628hc3a3a3f2a,
):
    if not np.isfinite(
        risk_628hc3a3a3f2a
    ).all():
        raise RuntimeError(
            "Selected uncertainty risk contains non-finite values."
        )

    if not np.all(
        (
            risk_628hc3a3a3f2a
            >=
            0.0
        )
        &
        (
            risk_628hc3a3a3f2a
            <=
            1.0
        )
    ):
        raise RuntimeError(
            "Selected uncertainty risk falls outside [0,1]."
        )

if len(manifest_628hc3a3a3f2a) != 315:
    raise RuntimeError(
        "Expected exactly 315 corruption scenarios."
    )

scenario_number_628hc3a3a3f2a = (
    manifest_628hc3a3a3f2a[
        "scenario_number"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

scenario_id_628hc3a3a3f2a = (
    manifest_628hc3a3a3f2a[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy(
        dtype="U"
    )
)

if not np.array_equal(
    scenario_number_628hc3a3a3f2a,
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Manifest scenario order is not exactly 1..315."
    )

# ---------------------------------------------------------------
# 3. Resolve the existing project freeze directory
# ---------------------------------------------------------------

cwd_628hc3a3a3f2a = Path.cwd()

if (cwd_628hc3a3a3f2a / "06_results").exists():
    project_root_628hc3a3a3f2a = (
        cwd_628hc3a3a3f2a
    )

elif (
    cwd_628hc3a3a3f2a.parent
    /
    "06_results"
).exists():
    project_root_628hc3a3a3f2a = (
        cwd_628hc3a3a3f2a.parent
    )

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc3a3a3f2a = (
    project_root_628hc3a3a3f2a
    /
    "06_results"
    /
    "step6_reliability_signals"
    /
    "fd002_c3a3"
)

freeze_dir_628hc3a3a3f2a.mkdir(
    parents=True,
    exist_ok=True,
)

risk_path_628hc3a3a3f2a = (
    freeze_dir_628hc3a3a3f2a
    /
    "fd002_c3a3_selected_uncertainty_reliability.npz"
)

decision_csv_path_628hc3a3a3f2a = (
    freeze_dir_628hc3a3a3f2a
    /
    "fd002_c3a3_adaptive_fusion_decision.csv"
)

metadata_path_628hc3a3a3f2a = (
    freeze_dir_628hc3a3a3f2a
    /
    "fd002_c3a3_selected_reliability_metadata.json"
)

existing_paths_628hc3a3a3f2a = [
    path_628hc3a3a3f2a
    for path_628hc3a3a3f2a in (
        risk_path_628hc3a3a3f2a,
        decision_csv_path_628hc3a3a3f2a,
        metadata_path_628hc3a3a3f2a,
    )
    if path_628hc3a3a3f2a.exists()
]

if existing_paths_628hc3a3a3f2a:
    raise FileExistsError(
        "Selected-reliability freeze artifact already exists. "
        "Nothing was overwritten:\n"
        +
        "\n".join(
            str(path_628hc3a3a3f2a)
            for path_628hc3a3a3f2a
            in existing_paths_628hc3a3a3f2a
        )
    )

# ---------------------------------------------------------------
# 4. SHA-256 helpers
# ---------------------------------------------------------------

def sha256_array_628hc3a3a3f2a(array):
    contiguous = np.ascontiguousarray(
        np.asarray(array)
    )

    return hashlib.sha256(
        contiguous.tobytes(
            order="C"
        )
    ).hexdigest()


def sha256_file_628hc3a3a3f2a(path):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_handle_628hc3a3a3f2a:

        while True:
            block_628hc3a3a3f2a = (
                file_handle_628hc3a3a3f2a.read(
                    1024
                    *
                    1024
                )
            )

            if not block_628hc3a3a3f2a:
                break

            digest.update(
                block_628hc3a3a3f2a
            )

    return digest.hexdigest()


corrupted_risk_hash_628hc3a3a3f2a = (
    sha256_array_628hc3a3a3f2a(
        corrupted_selected_risk_628hc3a3a3f2a
    )
)

clean_risk_hash_628hc3a3a3f2a = (
    sha256_array_628hc3a3a3f2a(
        clean_selected_risk_628hc3a3a3f2a
    )
)

# ---------------------------------------------------------------
# 5. Write the selected reliability-risk artifact
# ---------------------------------------------------------------

np.savez_compressed(
    risk_path_628hc3a3a3f2a,

    corrupted_uncertainty_risk=
        corrupted_selected_risk_628hc3a3a3f2a,

    clean_uncertainty_risk=
        clean_selected_risk_628hc3a3a3f2a,

    scenario_number=
        scenario_number_628hc3a3a3f2a,

    scenario_id=
        scenario_id_628hc3a3a3f2a,
)

# ---------------------------------------------------------------
# 6. Write the exact A7 decision table
# ---------------------------------------------------------------

decision_table_628hc3a3a3f2a.to_csv(
    decision_csv_path_628hc3a3a3f2a,
    index=False,
)

risk_file_hash_628hc3a3a3f2a = (
    sha256_file_628hc3a3a3f2a(
        risk_path_628hc3a3a3f2a
    )
)

decision_file_hash_628hc3a3a3f2a = (
    sha256_file_628hc3a3a3f2a(
        decision_csv_path_628hc3a3a3f2a
    )
)

# ---------------------------------------------------------------
# 7. Write freeze metadata
# ---------------------------------------------------------------

metadata_628hc3a3a3f2a = {
    "dataset":
        "NASA C-MAPSS FD002",

    "step":
        "6.28H-C3-A3-A3-F2-A",

    "development_decision_source_step":
        "6.28H-C3-A3-A3-A7",

    "freeze_status":
        "WRITE_COMPLETE_RELOAD_NOT_YET_VERIFIED",

    "selected_reliability_architecture":
        "uncertainty_only",

    "decision":
        "retain_uncertainty_only",

    "adaptive_fusion_accepted":
        False,

    "adaptive_candidates_evaluated":
        3,

    "adaptive_candidates_passing_all_goals":
        0,

    "selected_risk_source_objects": {
        "corrupted":
            "FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1",

        "clean":
            "FD002_DEV_SELECTIVE_RISK_628FCR2",
    },

    "selected_risk_shape": {
        "corrupted": [
            315,
            8367,
        ],

        "clean": [
            8367,
        ],
    },

    "array_sha256": {
        "corrupted_uncertainty_risk":
            corrupted_risk_hash_628hc3a3a3f2a,

        "clean_uncertainty_risk":
            clean_risk_hash_628hc3a3a3f2a,
    },

    "file_sha256": {
        "selected_risk_npz":
            risk_file_hash_628hc3a3a3f2a,

        "decision_csv":
            decision_file_hash_628hc3a3a3f2a,
    },

    "methodological_status": {
        "dev_rul_labels_used_to_make_development_decision":
            True,

        "official_test_labels_read":
            False,

        "test_scored":
            False,

        "new_architecture_introduced":
            False,

        "new_gamma_selected":
            False,

        "new_gate_threshold_selected":
            False,

        "abstention_threshold_selected":
            False,

        "model_executed":
            False,

        "model_retrained":
            False,
    },
}

with open(
    metadata_path_628hc3a3a3f2a,
    "w",
    encoding="utf-8",
) as metadata_file_628hc3a3a3f2a:

    json.dump(
        metadata_628hc3a3a3f2a,
        metadata_file_628hc3a3a3f2a,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------
# 8. Write-only closure
# ---------------------------------------------------------------

written_paths_628hc3a3a3f2a = [
    risk_path_628hc3a3a3f2a,
    decision_csv_path_628hc3a3a3f2a,
    metadata_path_628hc3a3a3f2a,
]

all_files_written_628hc3a3a3f2a = bool(
    all(
        path_628hc3a3a3f2a.exists()
        and
        path_628hc3a3a3f2a.stat().st_size
        >
        0
        for path_628hc3a3a3f2a
        in written_paths_628hc3a3a3f2a
    )
)

if not all_files_written_628hc3a3a3f2a:
    raise RuntimeError(
        "One or more F2-A freeze artifacts were not written correctly."
    )

print("\n--- FROZEN DEVELOPMENT DECISION ---")

print(
    "Selected reliability architecture :",
    selected_architecture_628hc3a3a3f2a,
)

print(
    "Decision                          :",
    decision_628hc3a3a3f2a,
)

print(
    "Adaptive fusion accepted          :",
    adaptive_accepted_628hc3a3a3f2a,
)

print("\n--- WRITTEN ARTIFACTS ---")

for path_628hc3a3a3f2a in (
    written_paths_628hc3a3a3f2a
):
    print(
        path_628hc3a3a3f2a.name,
        "| exists =",
        path_628hc3a3a3f2a.exists(),
        "| bytes =",
        path_628hc3a3a3f2a.stat().st_size,
    )

print("\n--- ARRAY SHA-256 ---")

print(
    "corrupted uncertainty risk :",
    corrupted_risk_hash_628hc3a3a3f2a,
)

print(
    "clean uncertainty risk     :",
    clean_risk_hash_628hc3a3a3f2a,
)

print("\n--- F2-A STATUS ---")

print(
    "All files written correctly     :",
    all_files_written_628hc3a3a3f2a,
)

print(
    "Independent reload verified     :",
    False,
)

print("DEV RUL labels used for decision : True")
print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Abstention threshold selected    : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files written                    : True")

print(
    "\nNext required step: "
    "F2-B independent reload and exact verification."
)

print(
    "\n=== END STEP 6.28H-C3-A3-A3-F2-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C3-A3-A3-F2-B
# INDEPENDENT RELOAD + EXACT SELECTED-RELIABILITY VERIFICATION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C3-A3-A3-F2-B — INDEPENDENT RELOAD + EXACT VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Resolve the frozen paths independently
# ---------------------------------------------------------------

cwd_628hc3a3a3f2b = Path.cwd()

if (cwd_628hc3a3a3f2b / "06_results").exists():
    project_root_628hc3a3a3f2b = cwd_628hc3a3a3f2b

elif (cwd_628hc3a3a3f2b.parent / "06_results").exists():
    project_root_628hc3a3a3f2b = cwd_628hc3a3a3f2b.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc3a3a3f2b = (
    project_root_628hc3a3a3f2b
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

risk_path_628hc3a3a3f2b = (
    freeze_dir_628hc3a3a3f2b
    / "fd002_c3a3_selected_uncertainty_reliability.npz"
)

decision_csv_path_628hc3a3a3f2b = (
    freeze_dir_628hc3a3a3f2b
    / "fd002_c3a3_adaptive_fusion_decision.csv"
)

metadata_path_628hc3a3a3f2b = (
    freeze_dir_628hc3a3a3f2b
    / "fd002_c3a3_selected_reliability_metadata.json"
)

for path_628hc3a3a3f2b in (
    risk_path_628hc3a3a3f2b,
    decision_csv_path_628hc3a3a3f2b,
    metadata_path_628hc3a3a3f2b,
):
    if not path_628hc3a3a3f2b.exists():
        raise FileNotFoundError(
            f"Required F2-A artifact is missing: {path_628hc3a3a3f2b}"
        )

# ---------------------------------------------------------------
# 2. Reload metadata + NPZ + decision CSV
# ---------------------------------------------------------------

with open(
    metadata_path_628hc3a3a3f2b,
    "r",
    encoding="utf-8",
) as metadata_file_628hc3a3a3f2b:

    metadata_628hc3a3a3f2b = json.load(
        metadata_file_628hc3a3a3f2b
    )

with np.load(
    risk_path_628hc3a3a3f2b,
    allow_pickle=False,
) as frozen_628hc3a3a3f2b:

    actual_keys_628hc3a3a3f2b = set(
        frozen_628hc3a3a3f2b.files
    )

    corrupted_reload_628hc3a3a3f2b = np.asarray(
        frozen_628hc3a3a3f2b[
            "corrupted_uncertainty_risk"
        ],
        dtype=np.float64,
    )

    clean_reload_628hc3a3a3f2b = np.asarray(
        frozen_628hc3a3a3f2b[
            "clean_uncertainty_risk"
        ],
        dtype=np.float64,
    ).reshape(-1)

    scenario_number_reload_628hc3a3a3f2b = np.asarray(
        frozen_628hc3a3a3f2b[
            "scenario_number"
        ],
        dtype=np.int64,
    )

    scenario_id_reload_628hc3a3a3f2b = np.asarray(
        frozen_628hc3a3a3f2b[
            "scenario_id"
        ]
    ).astype(str)

decision_reload_628hc3a3a3f2b = pd.read_csv(
    decision_csv_path_628hc3a3a3f2b
)

# ---------------------------------------------------------------
# 3. NPZ structural contract
# ---------------------------------------------------------------

expected_keys_628hc3a3a3f2b = {
    "corrupted_uncertainty_risk",
    "clean_uncertainty_risk",
    "scenario_number",
    "scenario_id",
}

keys_exact_628hc3a3a3f2b = bool(
    actual_keys_628hc3a3a3f2b
    ==
    expected_keys_628hc3a3a3f2b
)

corrupted_shape_exact_628hc3a3a3f2b = bool(
    corrupted_reload_628hc3a3a3f2b.shape
    ==
    (315, 8367)
)

clean_shape_exact_628hc3a3a3f2b = bool(
    clean_reload_628hc3a3a3f2b.shape
    ==
    (8367,)
)

values_valid_628hc3a3a3f2b = bool(
    np.isfinite(
        corrupted_reload_628hc3a3a3f2b
    ).all()

    and

    np.isfinite(
        clean_reload_628hc3a3a3f2b
    ).all()

    and

    np.all(
        (
            corrupted_reload_628hc3a3a3f2b
            >=
            0.0
        )
        &
        (
            corrupted_reload_628hc3a3a3f2b
            <=
            1.0
        )
    )

    and

    np.all(
        (
            clean_reload_628hc3a3a3f2b
            >=
            0.0
        )
        &
        (
            clean_reload_628hc3a3a3f2b
            <=
            1.0
        )
    )
)

# ---------------------------------------------------------------
# 4. Scenario-order contract
# ---------------------------------------------------------------

manifest_628hc3a3a3f2b = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

scenario_number_memory_628hc3a3a3f2b = (
    manifest_628hc3a3a3f2b[
        "scenario_number"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

scenario_id_memory_628hc3a3a3f2b = (
    manifest_628hc3a3a3f2b[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)

scenario_number_exact_628hc3a3a3f2b = bool(
    np.array_equal(
        scenario_number_reload_628hc3a3a3f2b,
        scenario_number_memory_628hc3a3a3f2b,
    )

    and

    np.array_equal(
        scenario_number_reload_628hc3a3a3f2b,
        np.arange(
            1,
            316,
            dtype=np.int64,
        ),
    )
)

scenario_id_exact_628hc3a3a3f2b = bool(
    np.array_equal(
        scenario_id_reload_628hc3a3a3f2b,
        scenario_id_memory_628hc3a3a3f2b,
    )
)

# ---------------------------------------------------------------
# 5. SHA-256 helpers
# ---------------------------------------------------------------

def sha256_array_628hc3a3a3f2b(array):

    contiguous_628hc3a3a3f2b = np.ascontiguousarray(
        np.asarray(array)
    )

    return hashlib.sha256(
        contiguous_628hc3a3a3f2b.tobytes(
            order="C"
        )
    ).hexdigest()


def sha256_file_628hc3a3a3f2b(path):

    digest_628hc3a3a3f2b = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_handle_628hc3a3a3f2b:

        while True:

            block_628hc3a3a3f2b = (
                file_handle_628hc3a3a3f2b.read(
                    1024 * 1024
                )
            )

            if not block_628hc3a3a3f2b:
                break

            digest_628hc3a3a3f2b.update(
                block_628hc3a3a3f2b
            )

    return digest_628hc3a3a3f2b.hexdigest()

# ---------------------------------------------------------------
# 6. Array-hash verification
# ---------------------------------------------------------------

corrupted_array_hash_exact_628hc3a3a3f2b = bool(
    sha256_array_628hc3a3a3f2b(
        corrupted_reload_628hc3a3a3f2b
    )
    ==
    metadata_628hc3a3a3f2b[
        "array_sha256"
    ][
        "corrupted_uncertainty_risk"
    ]
)

clean_array_hash_exact_628hc3a3a3f2b = bool(
    sha256_array_628hc3a3a3f2b(
        clean_reload_628hc3a3a3f2b
    )
    ==
    metadata_628hc3a3a3f2b[
        "array_sha256"
    ][
        "clean_uncertainty_risk"
    ]
)

# ---------------------------------------------------------------
# 7. File-hash verification
# ---------------------------------------------------------------

risk_file_hash_exact_628hc3a3a3f2b = bool(
    sha256_file_628hc3a3a3f2b(
        risk_path_628hc3a3a3f2b
    )
    ==
    metadata_628hc3a3a3f2b[
        "file_sha256"
    ][
        "selected_risk_npz"
    ]
)

decision_file_hash_exact_628hc3a3a3f2b = bool(
    sha256_file_628hc3a3a3f2b(
        decision_csv_path_628hc3a3a3f2b
    )
    ==
    metadata_628hc3a3a3f2b[
        "file_sha256"
    ][
        "decision_csv"
    ]
)

# ---------------------------------------------------------------
# 8. Exact reload-vs-memory verification
# ---------------------------------------------------------------

corrupted_memory_exact_628hc3a3a3f2b = bool(
    np.array_equal(
        corrupted_reload_628hc3a3a3f2b,

        np.asarray(
            FD002_CORRUPTED_UNCERTAINTY_RISK_MATRIX_628HC3A0R1,
            dtype=np.float64,
        ),
    )
)

clean_memory_exact_628hc3a3a3f2b = bool(
    np.array_equal(
        clean_reload_628hc3a3a3f2b,

        np.asarray(
            FD002_DEV_SELECTIVE_RISK_628FCR2,
            dtype=np.float64,
        ).reshape(-1),
    )
)

# ---------------------------------------------------------------
# 9. Decision-table semantic verification
#
# CSV floating values are checked at 1e-12 tolerance.
# Boolean/candidate fields are checked exactly.
# ---------------------------------------------------------------

decision_memory_628hc3a3a3f2b = (
    FD002_C3A3A3A7_ADAPTIVE_DECISION
    .copy()
    .reset_index(drop=True)
)

decision_columns_exact_628hc3a3a3f2b = bool(
    list(
        decision_reload_628hc3a3a3f2b.columns
    )
    ==
    list(
        decision_memory_628hc3a3a3f2b.columns
    )
)

decision_rows_exact_628hc3a3a3f2b = bool(
    len(
        decision_reload_628hc3a3a3f2b
    )
    ==
    len(
        decision_memory_628hc3a3a3f2b
    )
    ==
    3
)

decision_candidates_exact_628hc3a3a3f2b = bool(
    decision_rows_exact_628hc3a3a3f2b

    and

    np.array_equal(
        decision_reload_628hc3a3a3f2b[
            "candidate"
        ]
        .astype(str)
        .to_numpy(),

        decision_memory_628hc3a3a3f2b[
            "candidate"
        ]
        .astype(str)
        .to_numpy(),
    )
)

decision_bool_columns_628hc3a3a3f2b = [
    "preserves_clean_90_80_70",
    "preserves_stuck_low",
    "repairs_stuck_medium_positive",
    "repairs_stuck_high_positive",
    "adaptive_acceptance_passed",
]

decision_booleans_exact_628hc3a3a3f2b = bool(
    all(
        np.array_equal(
            decision_reload_628hc3a3a3f2b[
                column_628hc3a3a3f2b
            ]
            .astype(bool)
            .to_numpy(),

            decision_memory_628hc3a3a3f2b[
                column_628hc3a3a3f2b
            ]
            .astype(bool)
            .to_numpy(),
        )

        for column_628hc3a3a3f2b
        in decision_bool_columns_628hc3a3a3f2b
    )
)

decision_numeric_columns_628hc3a3a3f2b = [
    "clean_gain90",
    "clean_gain80",
    "clean_gain70",
    "stuck_low_gain80",
    "stuck_medium_gain80",
    "stuck_high_gain80",
]

decision_numerics_exact_628hc3a3a3f2b = bool(
    all(
        np.allclose(
            decision_reload_628hc3a3a3f2b[
                column_628hc3a3a3f2b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            decision_memory_628hc3a3a3f2b[
                column_628hc3a3a3f2b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            rtol=0.0,
            atol=1e-12,
        )

        for column_628hc3a3a3f2b
        in decision_numeric_columns_628hc3a3a3f2b
    )
)

decision_semantic_exact_628hc3a3a3f2b = bool(
    decision_columns_exact_628hc3a3a3f2b
    and
    decision_rows_exact_628hc3a3a3f2b
    and
    decision_candidates_exact_628hc3a3a3f2b
    and
    decision_booleans_exact_628hc3a3a3f2b
    and
    decision_numerics_exact_628hc3a3a3f2b
)

# ---------------------------------------------------------------
# 10. Metadata-contract verification
# ---------------------------------------------------------------

metadata_contract_exact_628hc3a3a3f2b = bool(

    metadata_628hc3a3a3f2b[
        "dataset"
    ]
    ==
    "NASA C-MAPSS FD002"

    and

    metadata_628hc3a3a3f2b[
        "development_decision_source_step"
    ]
    ==
    "6.28H-C3-A3-A3-A7"

    and

    metadata_628hc3a3a3f2b[
        "selected_reliability_architecture"
    ]
    ==
    "uncertainty_only"

    and

    metadata_628hc3a3a3f2b[
        "decision"
    ]
    ==
    "retain_uncertainty_only"

    and

    metadata_628hc3a3a3f2b[
        "adaptive_fusion_accepted"
    ]
    is False

    and

    metadata_628hc3a3a3f2b[
        "adaptive_candidates_evaluated"
    ]
    ==
    3

    and

    metadata_628hc3a3a3f2b[
        "adaptive_candidates_passing_all_goals"
    ]
    ==
    0

    and

    metadata_628hc3a3a3f2b[
        "selected_risk_shape"
    ][
        "corrupted"
    ]
    ==
    [
        315,
        8367,
    ]

    and

    metadata_628hc3a3a3f2b[
        "selected_risk_shape"
    ][
        "clean"
    ]
    ==
    [
        8367,
    ]

    and

    metadata_628hc3a3a3f2b[
        "methodological_status"
    ][
        "official_test_labels_read"
    ]
    is False

    and

    metadata_628hc3a3a3f2b[
        "methodological_status"
    ][
        "test_scored"
    ]
    is False

    and

    metadata_628hc3a3a3f2b[
        "methodological_status"
    ][
        "abstention_threshold_selected"
    ]
    is False
)

# ---------------------------------------------------------------
# 11. Publish verified in-memory frozen objects
# ---------------------------------------------------------------

FD002_FROZEN_SELECTED_CORRUPTED_RISK_628HC3A3A3F2B = (
    corrupted_reload_628hc3a3a3f2b
)

FD002_FROZEN_SELECTED_CLEAN_RISK_628HC3A3A3F2B = (
    clean_reload_628hc3a3a3f2b
)

FD002_FROZEN_ADAPTIVE_DECISION_628HC3A3A3F2B = (
    decision_reload_628hc3a3a3f2b
)

# ---------------------------------------------------------------
# 12. Independent reload audit
# ---------------------------------------------------------------

audit_628hc3a3a3f2b = pd.DataFrame(
    {
        "check": [
            "npz_keys_exact",
            "corrupted_shape_exact",
            "clean_shape_exact",
            "all_risk_values_valid",
            "scenario_number_order_exact",
            "scenario_id_order_exact",
            "corrupted_array_hash_exact",
            "clean_array_hash_exact",
            "risk_file_hash_exact",
            "decision_file_hash_exact",
            "corrupted_exact_vs_memory",
            "clean_exact_vs_memory",
            "decision_semantic_exact",
            "metadata_contract_exact",
        ],

        "passed": [
            keys_exact_628hc3a3a3f2b,
            corrupted_shape_exact_628hc3a3a3f2b,
            clean_shape_exact_628hc3a3a3f2b,
            values_valid_628hc3a3a3f2b,
            scenario_number_exact_628hc3a3a3f2b,
            scenario_id_exact_628hc3a3a3f2b,
            corrupted_array_hash_exact_628hc3a3a3f2b,
            clean_array_hash_exact_628hc3a3a3f2b,
            risk_file_hash_exact_628hc3a3a3f2b,
            decision_file_hash_exact_628hc3a3a3f2b,
            corrupted_memory_exact_628hc3a3a3f2b,
            clean_memory_exact_628hc3a3a3f2b,
            decision_semantic_exact_628hc3a3a3f2b,
            metadata_contract_exact_628hc3a3a3f2b,
        ],
    }
)

print(
    "\n--- F2-B INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc3a3a3f2b
)

all_checks_passed_628hc3a3a3f2b = bool(
    audit_628hc3a3a3f2b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc3a3a3f2b:
    raise RuntimeError(
        "F2-B selected-reliability verification failed."
    )

# ---------------------------------------------------------------
# 13. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C3-A3-A3-F2-B FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Selected reliability architecture :",
    metadata_628hc3a3a3f2b[
        "selected_reliability_architecture"
    ],
)

print(
    "Decision                          :",
    metadata_628hc3a3a3f2b[
        "decision"
    ],
)

print(
    "Corrupted risk shape              :",
    corrupted_reload_628hc3a3a3f2b.shape,
)

print(
    "Clean risk shape                  :",
    clean_reload_628hc3a3a3f2b.shape,
)

print(
    "Both array hashes exact           :",
    bool(
        corrupted_array_hash_exact_628hc3a3a3f2b
        and
        clean_array_hash_exact_628hc3a3a3f2b
    ),
)

print(
    "Both file hashes exact            :",
    bool(
        risk_file_hash_exact_628hc3a3a3f2b
        and
        decision_file_hash_exact_628hc3a3a3f2b
    ),
)

print(
    "Both arrays exact vs memory       :",
    bool(
        corrupted_memory_exact_628hc3a3a3f2b
        and
        clean_memory_exact_628hc3a3a3f2b
    ),
)

print(
    "Decision table exact              :",
    decision_semantic_exact_628hc3a3a3f2b,
)

print(
    "Metadata contract exact           :",
    metadata_contract_exact_628hc3a3a3f2b,
)

print(
    "Independent reload verified       :",
    all_checks_passed_628hc3a3a3f2b,
)

print("DEV RUL labels used in this step   : False")
print("Official TEST labels read          : False")
print("TEST scored                        : False")
print("Abstention threshold selected      : False")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files modified                     : False")

print(
    "\n=== END STEP 6.28H-C3-A3-A3-F2-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C4-A0
# FIXED-THRESHOLD ABSTENTION OPERATING-POLICY TRANSFER DIAGNOSTIC
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C4-A0 — FIXED-THRESHOLD ABSTENTION POLICY TRANSFER")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Use only the independently verified frozen reliability risks
# ---------------------------------------------------------------

clean_risk_628hc4a0 = np.asarray(
    FD002_FROZEN_SELECTED_CLEAN_RISK_628HC3A3A3F2B,
    dtype=np.float64,
).reshape(-1)

corrupted_risk_628hc4a0 = np.asarray(
    FD002_FROZEN_SELECTED_CORRUPTED_RISK_628HC3A3A3F2B,
    dtype=np.float64,
)

clean_prediction_628hc4a0 = np.asarray(
    DEV_PREDICTIONS_FD002_628EB,
    dtype=np.float64,
).reshape(-1)

corrupted_prediction_628hc4a0 = np.asarray(
    FD002_CORRUPTED_DEV_PREDICTIONS_628HB,
    dtype=np.float64,
)

y_dev_628hc4a0 = np.asarray(
    Y_DEV_PRIMARY_FD002_628D,
    dtype=np.float64,
).reshape(-1)

manifest_628hc4a0 = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if clean_risk_628hc4a0.shape != (8367,):
    raise RuntimeError(
        f"Unexpected clean risk shape: "
        f"{clean_risk_628hc4a0.shape}"
    )

if corrupted_risk_628hc4a0.shape != (315, 8367):
    raise RuntimeError(
        f"Unexpected corrupted risk shape: "
        f"{corrupted_risk_628hc4a0.shape}"
    )

if clean_prediction_628hc4a0.shape != (8367,):
    raise RuntimeError(
        f"Unexpected clean prediction shape: "
        f"{clean_prediction_628hc4a0.shape}"
    )

if corrupted_prediction_628hc4a0.shape != (315, 8367):
    raise RuntimeError(
        f"Unexpected corrupted prediction shape: "
        f"{corrupted_prediction_628hc4a0.shape}"
    )

if y_dev_628hc4a0.shape != (8367,):
    raise RuntimeError(
        f"Unexpected DEV target shape: "
        f"{y_dev_628hc4a0.shape}"
    )

if len(manifest_628hc4a0) != 315:
    raise RuntimeError(
        "Expected exactly 315 corruption scenarios."
    )

if not np.array_equal(
    manifest_628hc4a0[
        "scenario_number"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        316,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Scenario order is not exactly 1..315."
    )

for values_628hc4a0 in (
    clean_risk_628hc4a0,
    corrupted_risk_628hc4a0,
):
    if not np.isfinite(
        values_628hc4a0
    ).all():
        raise RuntimeError(
            "Frozen reliability risk contains non-finite values."
        )

    if not np.all(
        (
            values_628hc4a0 >= 0.0
        )
        &
        (
            values_628hc4a0 <= 1.0
        )
    ):
        raise RuntimeError(
            "Frozen reliability risk falls outside [0,1]."
        )

# ---------------------------------------------------------------
# 2. Candidate nominal operating coverages
#
# No coverage is selected here.
# ---------------------------------------------------------------

TARGET_COVERAGES_628HC4A0 = (
    0.90,
    0.80,
    0.70,
)

# ---------------------------------------------------------------
# 3. Deterministic clean-DEV threshold derivation
#
# Select the smallest observed clean-risk threshold that gives
# at least the requested clean coverage.
#
# This uses reliability scores only — NO RUL labels.
# ---------------------------------------------------------------

def threshold_for_minimum_coverage_628hc4a0(
    risk_values,
    target_coverage,
):
    risk_values = np.asarray(
        risk_values,
        dtype=np.float64,
    ).reshape(-1)

    n_628hc4a0 = len(
        risk_values
    )

    k_628hc4a0 = int(
        np.ceil(
            float(target_coverage)
            *
            n_628hc4a0
        )
    )

    if not (
        1
        <=
        k_628hc4a0
        <=
        n_628hc4a0
    ):
        raise RuntimeError(
            "Invalid target-coverage order statistic."
        )

    sorted_risk_628hc4a0 = np.sort(
        risk_values,
        kind="mergesort",
    )

    threshold_628hc4a0 = float(
        sorted_risk_628hc4a0[
            k_628hc4a0 - 1
        ]
    )

    realized_coverage_628hc4a0 = float(
        np.mean(
            risk_values
            <=
            threshold_628hc4a0
        )
    )

    if (
        realized_coverage_628hc4a0
        +
        1e-15
        <
        float(target_coverage)
    ):
        raise RuntimeError(
            "Derived threshold does not achieve target coverage."
        )

    return (
        threshold_628hc4a0,
        realized_coverage_628hc4a0,
        k_628hc4a0,
    )


threshold_rows_628hc4a0 = []

for target_coverage_628hc4a0 in (
    TARGET_COVERAGES_628HC4A0
):

    (
        threshold_628hc4a0,
        clean_coverage_628hc4a0,
        order_statistic_628hc4a0,
    ) = (
        threshold_for_minimum_coverage_628hc4a0(
            clean_risk_628hc4a0,
            target_coverage_628hc4a0,
        )
    )

    threshold_rows_628hc4a0.append(
        {
            "target_coverage":
                float(
                    target_coverage_628hc4a0
                ),

            "risk_threshold":
                threshold_628hc4a0,

            "clean_realized_coverage":
                clean_coverage_628hc4a0,

            "clean_accepted_windows":
                int(
                    np.sum(
                        clean_risk_628hc4a0
                        <=
                        threshold_628hc4a0
                    )
                ),

            "order_statistic_k":
                order_statistic_628hc4a0,
        }
    )

FD002_C4A0_THRESHOLDS_628HC4A0 = (
    pd.DataFrame(
        threshold_rows_628hc4a0
    )
)

print(
    "\n--- CLEAN-DEV FIXED THRESHOLD CONTRACT ---"
)

display(
    FD002_C4A0_THRESHOLDS_628HC4A0
)

# ---------------------------------------------------------------
# 4. Clean DEV policy evaluation
# ---------------------------------------------------------------

clean_error_628hc4a0 = np.abs(
    clean_prediction_628hc4a0
    -
    y_dev_628hc4a0
)

clean_full_mae_628hc4a0 = float(
    np.mean(
        clean_error_628hc4a0
    )
)

clean_policy_rows_628hc4a0 = []

for threshold_row_628hc4a0 in (
    threshold_rows_628hc4a0
):

    target_coverage_628hc4a0 = float(
        threshold_row_628hc4a0[
            "target_coverage"
        ]
    )

    threshold_628hc4a0 = float(
        threshold_row_628hc4a0[
            "risk_threshold"
        ]
    )

    accepted_628hc4a0 = (
        clean_risk_628hc4a0
        <=
        threshold_628hc4a0
    )

    accepted_count_628hc4a0 = int(
        accepted_628hc4a0.sum()
    )

    if accepted_count_628hc4a0 == 0:
        raise RuntimeError(
            "Clean policy accepted zero windows."
        )

    realized_coverage_628hc4a0 = float(
        np.mean(
            accepted_628hc4a0
        )
    )

    selective_mae_628hc4a0 = float(
        np.mean(
            clean_error_628hc4a0[
                accepted_628hc4a0
            ]
        )
    )

    gain_628hc4a0 = float(
        clean_full_mae_628hc4a0
        -
        selective_mae_628hc4a0
    )

    clean_policy_rows_628hc4a0.append(
        {
            "target_coverage":
                target_coverage_628hc4a0,

            "risk_threshold":
                threshold_628hc4a0,

            "realized_coverage":
                realized_coverage_628hc4a0,

            "accepted_windows":
                accepted_count_628hc4a0,

            "full_mae":
                clean_full_mae_628hc4a0,

            "selective_mae":
                selective_mae_628hc4a0,

            "gain":
                gain_628hc4a0,
        }
    )

FD002_C4A0_CLEAN_POLICY_628HC4A0 = (
    pd.DataFrame(
        clean_policy_rows_628hc4a0
    )
)

print(
    "\n--- CLEAN DEV FIXED-THRESHOLD PERFORMANCE ---"
)

display(
    FD002_C4A0_CLEAN_POLICY_628HC4A0
)

# ---------------------------------------------------------------
# 5. Transfer each fixed threshold unchanged to all 315 scenarios
# ---------------------------------------------------------------

scenario_rows_628hc4a0 = []

for threshold_row_628hc4a0 in (
    threshold_rows_628hc4a0
):

    target_coverage_628hc4a0 = float(
        threshold_row_628hc4a0[
            "target_coverage"
        ]
    )

    threshold_628hc4a0 = float(
        threshold_row_628hc4a0[
            "risk_threshold"
        ]
    )

    for scenario_index_628hc4a0 in range(
        315
    ):

        scenario_risk_628hc4a0 = (
            corrupted_risk_628hc4a0[
                scenario_index_628hc4a0
            ]
        )

        scenario_error_628hc4a0 = np.abs(
            corrupted_prediction_628hc4a0[
                scenario_index_628hc4a0
            ]
            -
            y_dev_628hc4a0
        )

        accepted_628hc4a0 = (
            scenario_risk_628hc4a0
            <=
            threshold_628hc4a0
        )

        accepted_count_628hc4a0 = int(
            accepted_628hc4a0.sum()
        )

        realized_coverage_628hc4a0 = float(
            np.mean(
                accepted_628hc4a0
            )
        )

        full_mae_628hc4a0 = float(
            np.mean(
                scenario_error_628hc4a0
            )
        )

        if accepted_count_628hc4a0 > 0:

            selective_mae_628hc4a0 = float(
                np.mean(
                    scenario_error_628hc4a0[
                        accepted_628hc4a0
                    ]
                )
            )

            gain_628hc4a0 = float(
                full_mae_628hc4a0
                -
                selective_mae_628hc4a0
            )

        else:

            selective_mae_628hc4a0 = np.nan
            gain_628hc4a0 = np.nan

        manifest_row_628hc4a0 = (
            manifest_628hc4a0.iloc[
                scenario_index_628hc4a0
            ]
        )

        scenario_rows_628hc4a0.append(
            {
                "target_coverage":
                    target_coverage_628hc4a0,

                "risk_threshold":
                    threshold_628hc4a0,

                "scenario_id":
                    str(
                        manifest_row_628hc4a0[
                            "scenario_id"
                        ]
                    ),

                "scenario_number":
                    int(
                        manifest_row_628hc4a0[
                            "scenario_number"
                        ]
                    ),

                "family":
                    str(
                        manifest_row_628hc4a0[
                            "family"
                        ]
                    ),

                "sensor":
                    str(
                        manifest_row_628hc4a0[
                            "sensor"
                        ]
                    ),

                "severity":
                    str(
                        manifest_row_628hc4a0[
                            "severity"
                        ]
                    ),

                "accepted_windows":
                    accepted_count_628hc4a0,

                "realized_coverage":
                    realized_coverage_628hc4a0,

                "full_mae":
                    full_mae_628hc4a0,

                "selective_mae":
                    selective_mae_628hc4a0,

                "gain":
                    gain_628hc4a0,
            }
        )

FD002_C4A0_SCENARIO_POLICY_628HC4A0 = (
    pd.DataFrame(
        scenario_rows_628hc4a0
    )
)

expected_scenario_rows_628hc4a0 = (
    315
    *
    len(
        TARGET_COVERAGES_628HC4A0
    )
)

if (
    len(
        FD002_C4A0_SCENARIO_POLICY_628HC4A0
    )
    !=
    expected_scenario_rows_628hc4a0
):
    raise RuntimeError(
        "Unexpected scenario-policy row count."
    )

# ---------------------------------------------------------------
# 6. Overall transfer summary
# ---------------------------------------------------------------

summary_rows_628hc4a0 = []

for target_coverage_628hc4a0 in (
    TARGET_COVERAGES_628HC4A0
):

    policy_df_628hc4a0 = (
        FD002_C4A0_SCENARIO_POLICY_628HC4A0[
            FD002_C4A0_SCENARIO_POLICY_628HC4A0[
                "target_coverage"
            ]
            ==
            target_coverage_628hc4a0
        ]
        .copy()
    )

    finite_gain_mask_628hc4a0 = np.isfinite(
        policy_df_628hc4a0[
            "gain"
        ].to_numpy(
            dtype=np.float64
        )
    )

    valid_policy_df_628hc4a0 = (
        policy_df_628hc4a0.loc[
            finite_gain_mask_628hc4a0
        ]
    )

    family_median_gain_628hc4a0 = (
        valid_policy_df_628hc4a0
        .groupby(
            "family",
            observed=True,
        )[
            "gain"
        ]
        .median()
    )

    gains_628hc4a0 = (
        valid_policy_df_628hc4a0[
            "gain"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    coverages_628hc4a0 = (
        policy_df_628hc4a0[
            "realized_coverage"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    summary_rows_628hc4a0.append(
        {
            "target_coverage":
                float(
                    target_coverage_628hc4a0
                ),

            "risk_threshold":
                float(
                    policy_df_628hc4a0[
                        "risk_threshold"
                    ]
                    .iloc[0]
                ),

            "median_realized_coverage":
                float(
                    np.median(
                        coverages_628hc4a0
                    )
                ),

            "q10_realized_coverage":
                float(
                    np.quantile(
                        coverages_628hc4a0,
                        0.10,
                    )
                ),

            "q90_realized_coverage":
                float(
                    np.quantile(
                        coverages_628hc4a0,
                        0.90,
                    )
                ),

            "min_realized_coverage":
                float(
                    np.min(
                        coverages_628hc4a0
                    )
                ),

            "median_gain":
                float(
                    np.median(
                        gains_628hc4a0
                    )
                ),

            "mean_gain":
                float(
                    np.mean(
                        gains_628hc4a0
                    )
                ),

            "q10_gain":
                float(
                    np.quantile(
                        gains_628hc4a0,
                        0.10,
                    )
                ),

            "worst_family_median_gain":
                float(
                    family_median_gain_628hc4a0.min()
                ),

            "positive_gain_scenarios":
                int(
                    np.sum(
                        gains_628hc4a0 > 0.0
                    )
                ),

            "negative_gain_scenarios":
                int(
                    np.sum(
                        gains_628hc4a0 < 0.0
                    )
                ),

            "zero_accepted_scenarios":
                int(
                    np.sum(
                        policy_df_628hc4a0[
                            "accepted_windows"
                        ]
                        .to_numpy(
                            dtype=np.int64
                        )
                        ==
                        0
                    )
                ),
        }
    )

FD002_C4A0_POLICY_SUMMARY_628HC4A0 = (
    pd.DataFrame(
        summary_rows_628hc4a0
    )
    .sort_values(
        "target_coverage",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "\n--- FIXED-THRESHOLD TRANSFER SUMMARY ---"
)

display(
    FD002_C4A0_POLICY_SUMMARY_628HC4A0
)

# ---------------------------------------------------------------
# 7. Family × severity transfer response
# ---------------------------------------------------------------

FD002_C4A0_FAMILY_SEVERITY_628HC4A0 = (
    FD002_C4A0_SCENARIO_POLICY_628HC4A0
    .groupby(
        [
            "target_coverage",
            "family",
            "severity",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        n_scenarios=(
            "scenario_number",
            "size",
        ),

        median_realized_coverage=(
            "realized_coverage",
            "median",
        ),

        median_gain=(
            "gain",
            "median",
        ),

        mean_gain=(
            "gain",
            "mean",
        ),
    )
)

severity_order_628hc4a0 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

FD002_C4A0_FAMILY_SEVERITY_628HC4A0[
    "_severity_order"
] = (
    FD002_C4A0_FAMILY_SEVERITY_628HC4A0[
        "severity"
    ]
    .map(
        severity_order_628hc4a0
    )
)

FD002_C4A0_FAMILY_SEVERITY_628HC4A0 = (
    FD002_C4A0_FAMILY_SEVERITY_628HC4A0
    .sort_values(
        [
            "target_coverage",
            "family",
            "_severity_order",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .drop(
        columns="_severity_order"
    )
    .reset_index(drop=True)
)

print(
    "\n--- FAMILY × SEVERITY FIXED-THRESHOLD RESPONSE ---"
)

display(
    FD002_C4A0_FAMILY_SEVERITY_628HC4A0
)

# ---------------------------------------------------------------
# 8. Audit
# ---------------------------------------------------------------

audit_628hc4a0 = pd.DataFrame(
    {
        "check": [
            "selected_clean_risk_shape_8367",
            "selected_corrupted_risk_shape_315x8367",
            "clean_prediction_shape_8367",
            "corrupted_prediction_shape_315x8367",
            "dev_labels_shape_8367",
            "manifest_rows_315",
            "three_candidate_coverages",
            "three_threshold_rows",
            "scenario_policy_rows_945",
            "all_thresholds_finite",
            "clean_coverages_meet_targets",
            "scenario_order_exact",
        ],

        "passed": [
            clean_risk_628hc4a0.shape
            ==
            (8367,),

            corrupted_risk_628hc4a0.shape
            ==
            (315, 8367),

            clean_prediction_628hc4a0.shape
            ==
            (8367,),

            corrupted_prediction_628hc4a0.shape
            ==
            (315, 8367),

            y_dev_628hc4a0.shape
            ==
            (8367,),

            len(
                manifest_628hc4a0
            )
            ==
            315,

            len(
                TARGET_COVERAGES_628HC4A0
            )
            ==
            3,

            len(
                FD002_C4A0_THRESHOLDS_628HC4A0
            )
            ==
            3,

            len(
                FD002_C4A0_SCENARIO_POLICY_628HC4A0
            )
            ==
            945,

            bool(
                np.isfinite(
                    FD002_C4A0_THRESHOLDS_628HC4A0[
                        "risk_threshold"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                )
                .all()
            ),

            bool(
                np.all(
                    FD002_C4A0_THRESHOLDS_628HC4A0[
                        "clean_realized_coverage"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                    +
                    1e-15
                    >=
                    FD002_C4A0_THRESHOLDS_628HC4A0[
                        "target_coverage"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                )
            ),

            bool(
                np.array_equal(
                    manifest_628hc4a0[
                        "scenario_number"
                    ]
                    .to_numpy(
                        dtype=np.int64
                    ),
                    np.arange(
                        1,
                        316,
                        dtype=np.int64,
                    ),
                )
            ),
        ],
    }
)

print(
    "\n--- C4-A0 AUDIT ---"
)

display(
    audit_628hc4a0
)

all_checks_passed_628hc4a0 = bool(
    audit_628hc4a0[
        "passed"
    ].all()
)

if not all_checks_passed_628hc4a0:
    raise RuntimeError(
        "C4-A0 operating-policy diagnostic failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C4-A0 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Reliability architecture            : uncertainty_only"
)

print(
    "Candidate nominal coverages         :",
    TARGET_COVERAGES_628HC4A0,
)

print(
    "Threshold source                    : clean DEV reliability distribution"
)

print(
    "Threshold derivation used RUL labels:",
    False,
)

print(
    "Operating policy selected           :",
    False,
)

print(
    "All checks passed                   :",
    all_checks_passed_628hc4a0,
)

print("DEV RUL labels used for evaluation  : True")
print("Official TEST labels read            : False")
print("TEST scored                          : False")
print("Abstention threshold selected        : False")
print("Model executed                       : False")
print("Model retrained                      : False")
print("Files written                        : False")

print(
    "\n=== END STEP 6.28H-C4-A0 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C4-A1
# ROBUST ABSTENTION OPERATING-POLICY SELECTION
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C4-A1 — ROBUST ABSTENTION OPERATING-POLICY SELECTION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reuse only the completed C4-A0 development results
# ---------------------------------------------------------------

summary_628hc4a1 = (
    FD002_C4A0_POLICY_SUMMARY_628HC4A0
    .copy()
    .sort_values(
        "target_coverage",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

clean_628hc4a1 = (
    FD002_C4A0_CLEAN_POLICY_628HC4A0
    .copy()
)

family_severity_628hc4a1 = (
    FD002_C4A0_FAMILY_SEVERITY_628HC4A0
    .copy()
)

thresholds_628hc4a1 = (
    FD002_C4A0_THRESHOLDS_628HC4A0
    .copy()
)

if len(summary_628hc4a1) != 3:
    raise RuntimeError(
        "Expected exactly three C4-A0 candidate policies."
    )

# ---------------------------------------------------------------
# 2. Pre-test operating-policy contract
#
# Practical coverage budget:
#     retain at least 80% nominal coverage
#     (no more than 20% nominal abstention)
#
# Robustness requirements:
#     positive clean gain
#     positive 10th-percentile scenario gain
#     positive worst-family median gain
#     no scenario with zero accepted windows
#     q10 realized coverage no more than 2 percentage points
#     below the nominal target
#
# Among eligible policies:
#     maximize q10 scenario gain
#     tie-break toward higher coverage
#
# No new threshold values are created here.
# ---------------------------------------------------------------

MIN_NOMINAL_COVERAGE_628HC4A1 = 0.80
MAX_Q10_COVERAGE_SHORTFALL_628HC4A1 = 0.02

selection_rows_628hc4a1 = []

for _, row_628hc4a1 in summary_628hc4a1.iterrows():

    target_628hc4a1 = float(
        row_628hc4a1[
            "target_coverage"
        ]
    )

    clean_match_628hc4a1 = (
        clean_628hc4a1[
            np.isclose(
                clean_628hc4a1[
                    "target_coverage"
                ].to_numpy(dtype=np.float64),
                target_628hc4a1,
                rtol=0.0,
                atol=1e-12,
            )
        ]
    )

    if len(clean_match_628hc4a1) != 1:
        raise RuntimeError(
            f"Clean-policy row missing or duplicated for "
            f"coverage {target_628hc4a1}."
        )

    clean_gain_628hc4a1 = float(
        clean_match_628hc4a1.iloc[0][
            "gain"
        ]
    )

    nominal_budget_ok_628hc4a1 = bool(
        target_628hc4a1
        >=
        MIN_NOMINAL_COVERAGE_628HC4A1
        -
        1e-12
    )

    clean_gain_positive_628hc4a1 = bool(
        clean_gain_628hc4a1 > 0.0
    )

    q10_gain_positive_628hc4a1 = bool(
        float(
            row_628hc4a1[
                "q10_gain"
            ]
        )
        >
        0.0
    )

    worst_family_positive_628hc4a1 = bool(
        float(
            row_628hc4a1[
                "worst_family_median_gain"
            ]
        )
        >
        0.0
    )

    no_zero_acceptance_628hc4a1 = bool(
        int(
            row_628hc4a1[
                "zero_accepted_scenarios"
            ]
        )
        ==
        0
    )

    q10_coverage_shortfall_628hc4a1 = float(
        target_628hc4a1
        -
        float(
            row_628hc4a1[
                "q10_realized_coverage"
            ]
        )
    )

    coverage_transfer_ok_628hc4a1 = bool(
        q10_coverage_shortfall_628hc4a1
        <=
        MAX_Q10_COVERAGE_SHORTFALL_628HC4A1
        +
        1e-12
    )

    eligible_628hc4a1 = bool(
        nominal_budget_ok_628hc4a1
        and
        clean_gain_positive_628hc4a1
        and
        q10_gain_positive_628hc4a1
        and
        worst_family_positive_628hc4a1
        and
        no_zero_acceptance_628hc4a1
        and
        coverage_transfer_ok_628hc4a1
    )

    selection_rows_628hc4a1.append(
        {
            "target_coverage":
                target_628hc4a1,

            "risk_threshold":
                float(
                    row_628hc4a1[
                        "risk_threshold"
                    ]
                ),

            "clean_gain":
                clean_gain_628hc4a1,

            "median_realized_coverage":
                float(
                    row_628hc4a1[
                        "median_realized_coverage"
                    ]
                ),

            "q10_realized_coverage":
                float(
                    row_628hc4a1[
                        "q10_realized_coverage"
                    ]
                ),

            "q10_coverage_shortfall":
                q10_coverage_shortfall_628hc4a1,

            "median_gain":
                float(
                    row_628hc4a1[
                        "median_gain"
                    ]
                ),

            "q10_gain":
                float(
                    row_628hc4a1[
                        "q10_gain"
                    ]
                ),

            "worst_family_median_gain":
                float(
                    row_628hc4a1[
                        "worst_family_median_gain"
                    ]
                ),

            "positive_gain_scenarios":
                int(
                    row_628hc4a1[
                        "positive_gain_scenarios"
                    ]
                ),

            "negative_gain_scenarios":
                int(
                    row_628hc4a1[
                        "negative_gain_scenarios"
                    ]
                ),

            "zero_accepted_scenarios":
                int(
                    row_628hc4a1[
                        "zero_accepted_scenarios"
                    ]
                ),

            "nominal_coverage_budget_ok":
                nominal_budget_ok_628hc4a1,

            "clean_gain_positive":
                clean_gain_positive_628hc4a1,

            "q10_gain_positive":
                q10_gain_positive_628hc4a1,

            "worst_family_median_positive":
                worst_family_positive_628hc4a1,

            "coverage_transfer_ok":
                coverage_transfer_ok_628hc4a1,

            "eligible":
                eligible_628hc4a1,
        }
    )

FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1 = (
    pd.DataFrame(
        selection_rows_628hc4a1
    )
)

print(
    "\n--- OPERATING-POLICY ELIGIBILITY AUDIT ---"
)

display(
    FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1
)

# ---------------------------------------------------------------
# 3. Select among already-defined eligible policies
# ---------------------------------------------------------------

eligible_628hc4a1 = (
    FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1[
        FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1[
            "eligible"
        ]
    ]
    .copy()
)

if len(eligible_628hc4a1) == 0:
    raise RuntimeError(
        "No C4-A0 operating policy satisfies the "
        "pre-test robustness contract."
    )

eligible_628hc4a1 = (
    eligible_628hc4a1
    .sort_values(
        [
            "q10_gain",
            "target_coverage",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

selected_row_628hc4a1 = (
    eligible_628hc4a1.iloc[0]
)

FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1 = float(
    selected_row_628hc4a1[
        "target_coverage"
    ]
)

FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1 = float(
    selected_row_628hc4a1[
        "risk_threshold"
    ]
)

FD002_C4A1_SELECTED_POLICY_NAME_628HC4A1 = (
    "uncertainty_only_fixed_threshold"
)

# ---------------------------------------------------------------
# 4. Verify selected threshold is exactly one of the A0 thresholds
# ---------------------------------------------------------------

threshold_match_628hc4a1 = (
    thresholds_628hc4a1[
        np.isclose(
            thresholds_628hc4a1[
                "target_coverage"
            ].to_numpy(dtype=np.float64),
            FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1,
            rtol=0.0,
            atol=1e-12,
        )
    ]
)

if len(threshold_match_628hc4a1) != 1:
    raise RuntimeError(
        "Selected C4-A0 threshold row is missing or duplicated."
    )

exact_a0_threshold_628hc4a1 = float(
    threshold_match_628hc4a1.iloc[0][
        "risk_threshold"
    ]
)

threshold_exact_from_a0_628hc4a1 = bool(
    FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1
    ==
    exact_a0_threshold_628hc4a1
)

if not threshold_exact_from_a0_628hc4a1:
    raise RuntimeError(
        "Selected threshold differs from the exact A0 threshold."
    )

# ---------------------------------------------------------------
# 5. Selected policy family × severity diagnostic
#
# Diagnostic only. This does NOT alter the selected policy.
# ---------------------------------------------------------------

selected_family_severity_628hc4a1 = (
    family_severity_628hc4a1[
        np.isclose(
            family_severity_628hc4a1[
                "target_coverage"
            ].to_numpy(dtype=np.float64),
            FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1,
            rtol=0.0,
            atol=1e-12,
        )
    ]
    .copy()
    .reset_index(drop=True)
)

FD002_C4A1_SELECTED_FAMILY_SEVERITY_628HC4A1 = (
    selected_family_severity_628hc4a1
)

FD002_C4A1_SELECTED_NEGATIVE_SUBGROUPS_628HC4A1 = (
    selected_family_severity_628hc4a1[
        selected_family_severity_628hc4a1[
            "median_gain"
        ]
        <
        0.0
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "\n--- SELECTED POLICY: FAMILY × SEVERITY RESPONSE ---"
)

display(
    FD002_C4A1_SELECTED_FAMILY_SEVERITY_628HC4A1
)

print(
    "\n--- SELECTED POLICY: NEGATIVE-MEDIAN SUBGROUPS ---"
)

display(
    FD002_C4A1_SELECTED_NEGATIVE_SUBGROUPS_628HC4A1
)

# ---------------------------------------------------------------
# 6. Compact selected-policy summary
# ---------------------------------------------------------------

selected_summary_628hc4a1 = pd.DataFrame(
    [
        {
            "policy":
                FD002_C4A1_SELECTED_POLICY_NAME_628HC4A1,

            "target_coverage":
                FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1,

            "risk_threshold":
                FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1,

            "clean_gain":
                float(
                    selected_row_628hc4a1[
                        "clean_gain"
                    ]
                ),

            "median_realized_coverage":
                float(
                    selected_row_628hc4a1[
                        "median_realized_coverage"
                    ]
                ),

            "q10_realized_coverage":
                float(
                    selected_row_628hc4a1[
                        "q10_realized_coverage"
                    ]
                ),

            "median_gain":
                float(
                    selected_row_628hc4a1[
                        "median_gain"
                    ]
                ),

            "q10_gain":
                float(
                    selected_row_628hc4a1[
                        "q10_gain"
                    ]
                ),

            "worst_family_median_gain":
                float(
                    selected_row_628hc4a1[
                        "worst_family_median_gain"
                    ]
                ),

            "positive_gain_scenarios":
                int(
                    selected_row_628hc4a1[
                        "positive_gain_scenarios"
                    ]
                ),

            "negative_gain_scenarios":
                int(
                    selected_row_628hc4a1[
                        "negative_gain_scenarios"
                    ]
                ),

            "negative_family_severity_groups":
                int(
                    len(
                        FD002_C4A1_SELECTED_NEGATIVE_SUBGROUPS_628HC4A1
                    )
                ),
        }
    ]
)

FD002_C4A1_SELECTED_POLICY_SUMMARY_628HC4A1 = (
    selected_summary_628hc4a1
)

print(
    "\n--- SELECTED OPERATING POLICY ---"
)

display(
    FD002_C4A1_SELECTED_POLICY_SUMMARY_628HC4A1
)

# ---------------------------------------------------------------
# 7. Integrity audit
# ---------------------------------------------------------------

audit_628hc4a1 = pd.DataFrame(
    {
        "check": [
            "three_A0_candidates_present",
            "minimum_nominal_coverage_080",
            "eligible_policy_exists",
            "selected_policy_is_eligible",
            "selected_threshold_exact_from_A0",
            "selected_architecture_remains_uncertainty_only",
            "no_new_threshold_generated",
            "test_untouched",
        ],

        "passed": [
            len(
                FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1
            )
            ==
            3,

            MIN_NOMINAL_COVERAGE_628HC4A1
            ==
            0.80,

            len(
                eligible_628hc4a1
            )
            >=
            1,

            bool(
                selected_row_628hc4a1[
                    "eligible"
                ]
            ),

            threshold_exact_from_a0_628hc4a1,

            FD002_C3A3A3A7_SELECTED_ARCHITECTURE
            ==
            "uncertainty_only",

            True,

            True,
        ],
    }
)

print(
    "\n--- C4-A1 AUDIT ---"
)

display(
    audit_628hc4a1
)

all_checks_passed_628hc4a1 = bool(
    audit_628hc4a1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc4a1:
    raise RuntimeError(
        "C4-A1 operating-policy selection audit failed."
    )

# ---------------------------------------------------------------
# 8. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C4-A1 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Reliability architecture          : uncertainty_only"
)

print(
    "Operating policy                 :",
    FD002_C4A1_SELECTED_POLICY_NAME_628HC4A1,
)

print(
    "Selected nominal coverage        :",
    FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1,
)

print(
    "Selected exact risk threshold    :",
    repr(
        FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1
    ),
)

print(
    "Selection objective              : maximize q10 gain among eligible policies"
)

print(
    "Minimum nominal coverage         :",
    MIN_NOMINAL_COVERAGE_628HC4A1,
)

print(
    "Negative family×severity groups  :",
    len(
        FD002_C4A1_SELECTED_NEGATIVE_SUBGROUPS_628HC4A1
    ),
)

print(
    "All checks passed                :",
    all_checks_passed_628hc4a1,
)

print("DEV RUL labels used for selection : True")
print("Official TEST labels read         : False")
print("TEST scored                       : False")
print("Abstention threshold selected     : True")
print("Model executed                    : False")
print("Model retrained                   : False")
print("Files written                     : False")

print(
    "\n=== END STEP 6.28H-C4-A1 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C4-F1-A
# FREEZE FD002 SELECTED ABSTENTION OPERATING POLICY
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C4-F1-A — FREEZE FD002 SELECTED ABSTENTION OPERATING POLICY")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Lock the completed C4-A1 selection
# ---------------------------------------------------------------

selected_policy_628hc4f1a = str(
    FD002_C4A1_SELECTED_POLICY_NAME_628HC4A1
)

selected_coverage_628hc4f1a = float(
    FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1
)

selected_threshold_628hc4f1a = float(
    FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1
)

selection_audit_628hc4f1a = (
    FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1
    .copy()
    .sort_values(
        "target_coverage",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

selected_summary_628hc4f1a = (
    FD002_C4A1_SELECTED_POLICY_SUMMARY_628HC4A1
    .copy()
    .reset_index(drop=True)
)

selected_family_severity_628hc4f1a = (
    FD002_C4A1_SELECTED_FAMILY_SEVERITY_628HC4A1
    .copy()
    .reset_index(drop=True)
)

negative_subgroups_628hc4f1a = (
    FD002_C4A1_SELECTED_NEGATIVE_SUBGROUPS_628HC4A1
    .copy()
    .reset_index(drop=True)
)

if selected_policy_628hc4f1a != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Unexpected selected operating-policy name."
    )

if selected_coverage_628hc4f1a != 0.80:
    raise RuntimeError(
        "Unexpected selected nominal coverage."
    )

if len(selected_summary_628hc4f1a) != 1:
    raise RuntimeError(
        "Selected policy summary must contain exactly one row."
    )

if len(selection_audit_628hc4f1a) != 3:
    raise RuntimeError(
        "Policy-selection audit must contain exactly three rows."
    )

if len(selected_family_severity_628hc4f1a) != 15:
    raise RuntimeError(
        "Selected family×severity response must contain exactly 15 rows."
    )

if len(negative_subgroups_628hc4f1a) != 2:
    raise RuntimeError(
        "Expected exactly two negative family×severity subgroups."
    )

negative_pairs_628hc4f1a = set(
    zip(
        negative_subgroups_628hc4f1a["family"].astype(str),
        negative_subgroups_628hc4f1a["severity"].astype(str),
    )
)

if negative_pairs_628hc4f1a != {
    ("stuck", "medium"),
    ("stuck", "high"),
}:
    raise RuntimeError(
        "Negative subgroup identity differs from the frozen C4-A1 result."
    )

selected_audit_row_628hc4f1a = (
    selection_audit_628hc4f1a[
        np.isclose(
            selection_audit_628hc4f1a[
                "target_coverage"
            ].to_numpy(dtype=np.float64),
            selected_coverage_628hc4f1a,
            rtol=0.0,
            atol=1e-12,
        )
    ]
)

if len(selected_audit_row_628hc4f1a) != 1:
    raise RuntimeError(
        "Selected coverage row is missing or duplicated in C4-A1 audit."
    )

if not bool(
    selected_audit_row_628hc4f1a.iloc[0]["eligible"]
):
    raise RuntimeError(
        "Selected operating policy is not marked eligible."
    )

threshold_exact_628hc4f1a = float(
    selected_audit_row_628hc4f1a.iloc[0][
        "risk_threshold"
    ]
)

if selected_threshold_628hc4f1a != threshold_exact_628hc4f1a:
    raise RuntimeError(
        "Selected threshold differs from the exact C4-A1 audit value."
    )

# ---------------------------------------------------------------
# 2. Resolve project directories + upstream frozen reliability
# ---------------------------------------------------------------

cwd_628hc4f1a = Path.cwd()

if (cwd_628hc4f1a / "06_results").exists():
    project_root_628hc4f1a = cwd_628hc4f1a

elif (cwd_628hc4f1a.parent / "06_results").exists():
    project_root_628hc4f1a = cwd_628hc4f1a.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

upstream_dir_628hc4f1a = (
    project_root_628hc4f1a
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

upstream_risk_path_628hc4f1a = (
    upstream_dir_628hc4f1a
    / "fd002_c3a3_selected_uncertainty_reliability.npz"
)

upstream_metadata_path_628hc4f1a = (
    upstream_dir_628hc4f1a
    / "fd002_c3a3_selected_reliability_metadata.json"
)

if not upstream_risk_path_628hc4f1a.exists():
    raise FileNotFoundError(
        "Frozen selected reliability-risk artifact is missing."
    )

if not upstream_metadata_path_628hc4f1a.exists():
    raise FileNotFoundError(
        "Frozen selected reliability metadata is missing."
    )

with open(
    upstream_metadata_path_628hc4f1a,
    "r",
    encoding="utf-8",
) as upstream_metadata_file_628hc4f1a:

    upstream_metadata_628hc4f1a = json.load(
        upstream_metadata_file_628hc4f1a
    )

if (
    upstream_metadata_628hc4f1a[
        "selected_reliability_architecture"
    ]
    !=
    "uncertainty_only"
):
    raise RuntimeError(
        "Upstream frozen reliability architecture is not uncertainty_only."
    )

# ---------------------------------------------------------------
# 3. File-hash helper
# ---------------------------------------------------------------

def sha256_file_628hc4f1a(path):

    digest_628hc4f1a = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_handle_628hc4f1a:

        while True:

            block_628hc4f1a = file_handle_628hc4f1a.read(
                1024 * 1024
            )

            if not block_628hc4f1a:
                break

            digest_628hc4f1a.update(
                block_628hc4f1a
            )

    return digest_628hc4f1a.hexdigest()


upstream_risk_hash_628hc4f1a = (
    sha256_file_628hc4f1a(
        upstream_risk_path_628hc4f1a
    )
)

expected_upstream_risk_hash_628hc4f1a = (
    upstream_metadata_628hc4f1a[
        "file_sha256"
    ][
        "selected_risk_npz"
    ]
)

if (
    upstream_risk_hash_628hc4f1a
    !=
    expected_upstream_risk_hash_628hc4f1a
):
    raise RuntimeError(
        "Upstream frozen reliability-risk file hash does not match metadata."
    )

# ---------------------------------------------------------------
# 4. Freeze directory + no-overwrite contract
# ---------------------------------------------------------------

freeze_dir_628hc4f1a = (
    project_root_628hc4f1a
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c4"
)

freeze_dir_628hc4f1a.mkdir(
    parents=True,
    exist_ok=True,
)

selection_csv_path_628hc4f1a = (
    freeze_dir_628hc4f1a
    / "fd002_c4_policy_selection_audit.csv"
)

family_csv_path_628hc4f1a = (
    freeze_dir_628hc4f1a
    / "fd002_c4_selected_family_severity_response.csv"
)

policy_json_path_628hc4f1a = (
    freeze_dir_628hc4f1a
    / "fd002_c4_selected_abstention_policy.json"
)

existing_paths_628hc4f1a = [
    path_628hc4f1a
    for path_628hc4f1a in (
        selection_csv_path_628hc4f1a,
        family_csv_path_628hc4f1a,
        policy_json_path_628hc4f1a,
    )
    if path_628hc4f1a.exists()
]

if existing_paths_628hc4f1a:
    raise FileExistsError(
        "C4 freeze artifact already exists. Nothing was overwritten:\n"
        +
        "\n".join(
            str(path_628hc4f1a)
            for path_628hc4f1a
            in existing_paths_628hc4f1a
        )
    )

# ---------------------------------------------------------------
# 5. Write exact development-selection tables
# ---------------------------------------------------------------

selection_audit_628hc4f1a.to_csv(
    selection_csv_path_628hc4f1a,
    index=False,
)

selected_family_severity_628hc4f1a.to_csv(
    family_csv_path_628hc4f1a,
    index=False,
)

selection_csv_hash_628hc4f1a = (
    sha256_file_628hc4f1a(
        selection_csv_path_628hc4f1a
    )
)

family_csv_hash_628hc4f1a = (
    sha256_file_628hc4f1a(
        family_csv_path_628hc4f1a
    )
)

# ---------------------------------------------------------------
# 6. Freeze selected operating-policy metadata
# ---------------------------------------------------------------

selected_summary_row_628hc4f1a = (
    selected_summary_628hc4f1a.iloc[0]
)

policy_metadata_628hc4f1a = {
    "dataset":
        "NASA C-MAPSS FD002",

    "step":
        "6.28H-C4-F1-A",

    "selection_source_step":
        "6.28H-C4-A1",

    "threshold_source_step":
        "6.28H-C4-A0",

    "upstream_reliability_freeze_step":
        "6.28H-C3-A3-A3-F2-B",

    "freeze_status":
        "WRITE_COMPLETE_RELOAD_NOT_YET_VERIFIED",

    "reliability_architecture":
        "uncertainty_only",

    "operating_policy":
        selected_policy_628hc4f1a,

    "acceptance_rule":
        "accept iff uncertainty_risk <= risk_threshold",

    "selected_nominal_coverage":
        selected_coverage_628hc4f1a,

    "selected_nominal_abstention":
        float(
            1.0
            -
            selected_coverage_628hc4f1a
        ),

    "selected_exact_risk_threshold":
        selected_threshold_628hc4f1a,

    "selection_contract": {
        "minimum_nominal_coverage":
            0.80,

        "maximum_q10_coverage_shortfall":
            0.02,

        "requires_positive_clean_gain":
            True,

        "requires_positive_q10_scenario_gain":
            True,

        "requires_positive_worst_family_median_gain":
            True,

        "requires_zero_zero_acceptance_scenarios":
            True,

        "objective":
            "maximize q10 scenario gain among eligible policies",

        "tie_break":
            "higher nominal coverage",
    },

    "selected_development_summary": {
        "clean_gain":
            float(
                selected_summary_row_628hc4f1a[
                    "clean_gain"
                ]
            ),

        "median_realized_coverage":
            float(
                selected_summary_row_628hc4f1a[
                    "median_realized_coverage"
                ]
            ),

        "q10_realized_coverage":
            float(
                selected_summary_row_628hc4f1a[
                    "q10_realized_coverage"
                ]
            ),

        "median_gain":
            float(
                selected_summary_row_628hc4f1a[
                    "median_gain"
                ]
            ),

        "q10_gain":
            float(
                selected_summary_row_628hc4f1a[
                    "q10_gain"
                ]
            ),

        "worst_family_median_gain":
            float(
                selected_summary_row_628hc4f1a[
                    "worst_family_median_gain"
                ]
            ),

        "positive_gain_scenarios":
            int(
                selected_summary_row_628hc4f1a[
                    "positive_gain_scenarios"
                ]
            ),

        "negative_gain_scenarios":
            int(
                selected_summary_row_628hc4f1a[
                    "negative_gain_scenarios"
                ]
            ),

        "negative_family_severity_groups":
            int(
                selected_summary_row_628hc4f1a[
                    "negative_family_severity_groups"
                ]
            ),
    },

    "known_negative_family_severity_groups": [
        {
            "family":
                str(
                    row_628hc4f1a[
                        "family"
                    ]
                ),

            "severity":
                str(
                    row_628hc4f1a[
                        "severity"
                    ]
                ),

            "median_realized_coverage":
                float(
                    row_628hc4f1a[
                        "median_realized_coverage"
                    ]
                ),

            "median_gain":
                float(
                    row_628hc4f1a[
                        "median_gain"
                    ]
                ),

            "mean_gain":
                float(
                    row_628hc4f1a[
                        "mean_gain"
                    ]
                ),
        }
        for _, row_628hc4f1a
        in negative_subgroups_628hc4f1a.iterrows()
    ],

    "upstream_reliability_artifact": {
        "filename":
            upstream_risk_path_628hc4f1a.name,

        "sha256":
            upstream_risk_hash_628hc4f1a,
    },

    "frozen_file_sha256": {
        "policy_selection_audit_csv":
            selection_csv_hash_628hc4f1a,

        "selected_family_severity_csv":
            family_csv_hash_628hc4f1a,
    },

    "methodological_status": {
        "dev_rul_labels_used_for_policy_selection":
            True,

        "official_test_labels_read":
            False,

        "test_scored":
            False,

        "abstention_threshold_selected":
            True,

        "threshold_selected_before_test":
            True,

        "model_executed":
            False,

        "model_retrained":
            False,
    },
}

with open(
    policy_json_path_628hc4f1a,
    "w",
    encoding="utf-8",
) as policy_json_file_628hc4f1a:

    json.dump(
        policy_metadata_628hc4f1a,
        policy_json_file_628hc4f1a,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------
# 7. Write-completion checks
# ---------------------------------------------------------------

written_paths_628hc4f1a = [
    selection_csv_path_628hc4f1a,
    family_csv_path_628hc4f1a,
    policy_json_path_628hc4f1a,
]

all_files_written_628hc4f1a = bool(
    all(
        path_628hc4f1a.exists()
        and
        path_628hc4f1a.stat().st_size > 0
        for path_628hc4f1a
        in written_paths_628hc4f1a
    )
)

if not all_files_written_628hc4f1a:
    raise RuntimeError(
        "One or more C4-F1-A artifacts were not written correctly."
    )

# ---------------------------------------------------------------
# 8. Final status
# ---------------------------------------------------------------

print("\n--- FROZEN OPERATING POLICY ---")

print(
    "Reliability architecture       : uncertainty_only"
)

print(
    "Operating policy               :",
    selected_policy_628hc4f1a,
)

print(
    "Selected nominal coverage      :",
    selected_coverage_628hc4f1a,
)

print(
    "Selected exact risk threshold  :",
    repr(
        selected_threshold_628hc4f1a
    ),
)

print(
    "Nominal abstention fraction    :",
    1.0
    -
    selected_coverage_628hc4f1a,
)

print(
    "Known negative subgroups       :",
    sorted(
        negative_pairs_628hc4f1a
    ),
)

print("\n--- WRITTEN ARTIFACTS ---")

for path_628hc4f1a in written_paths_628hc4f1a:
    print(
        path_628hc4f1a.name,
        "| exists =",
        path_628hc4f1a.exists(),
        "| bytes =",
        path_628hc4f1a.stat().st_size,
    )

print("\n--- F1-A STATUS ---")

print(
    "Upstream reliability hash exact :",
    upstream_risk_hash_628hc4f1a
    ==
    expected_upstream_risk_hash_628hc4f1a,
)

print(
    "All files written correctly     :",
    all_files_written_628hc4f1a,
)

print(
    "Independent reload verified     :",
    False,
)

print("DEV RUL labels used for selection : True")
print("Official TEST labels read         : False")
print("TEST scored                        : False")
print("Abstention threshold selected      : True")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files written                      : True")

print(
    "\nNext required step: "
    "C4-F1-B independent reload and exact verification."
)

print(
    "\n=== END STEP 6.28H-C4-F1-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C4-F1-B
# INDEPENDENT RELOAD + EXACT OPERATING-POLICY VERIFICATION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C4-F1-B — INDEPENDENT RELOAD + EXACT POLICY VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Resolve frozen C4 artifact paths independently
# ---------------------------------------------------------------

cwd_628hc4f1b = Path.cwd()

if (cwd_628hc4f1b / "06_results").exists():
    project_root_628hc4f1b = cwd_628hc4f1b

elif (cwd_628hc4f1b.parent / "06_results").exists():
    project_root_628hc4f1b = cwd_628hc4f1b.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

freeze_dir_628hc4f1b = (
    project_root_628hc4f1b
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c4"
)

selection_csv_path_628hc4f1b = (
    freeze_dir_628hc4f1b
    / "fd002_c4_policy_selection_audit.csv"
)

family_csv_path_628hc4f1b = (
    freeze_dir_628hc4f1b
    / "fd002_c4_selected_family_severity_response.csv"
)

policy_json_path_628hc4f1b = (
    freeze_dir_628hc4f1b
    / "fd002_c4_selected_abstention_policy.json"
)

required_paths_628hc4f1b = (
    selection_csv_path_628hc4f1b,
    family_csv_path_628hc4f1b,
    policy_json_path_628hc4f1b,
)

for path_628hc4f1b in required_paths_628hc4f1b:
    if not path_628hc4f1b.exists():
        raise FileNotFoundError(
            f"Required C4-F1-A artifact is missing: {path_628hc4f1b}"
        )

# ---------------------------------------------------------------
# 2. Independently reload the frozen artifacts
# ---------------------------------------------------------------

selection_reload_628hc4f1b = pd.read_csv(
    selection_csv_path_628hc4f1b
)

family_reload_628hc4f1b = pd.read_csv(
    family_csv_path_628hc4f1b
)

with open(
    policy_json_path_628hc4f1b,
    "r",
    encoding="utf-8",
) as policy_file_628hc4f1b:

    policy_reload_628hc4f1b = json.load(
        policy_file_628hc4f1b
    )

# ---------------------------------------------------------------
# 3. File-hash helper
# ---------------------------------------------------------------

def sha256_file_628hc4f1b(path):

    digest_628hc4f1b = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_handle_628hc4f1b:

        while True:

            block_628hc4f1b = file_handle_628hc4f1b.read(
                1024 * 1024
            )

            if not block_628hc4f1b:
                break

            digest_628hc4f1b.update(
                block_628hc4f1b
            )

    return digest_628hc4f1b.hexdigest()

# ---------------------------------------------------------------
# 4. Verify the two frozen CSV file hashes
# ---------------------------------------------------------------

selection_csv_hash_reload_628hc4f1b = (
    sha256_file_628hc4f1b(
        selection_csv_path_628hc4f1b
    )
)

family_csv_hash_reload_628hc4f1b = (
    sha256_file_628hc4f1b(
        family_csv_path_628hc4f1b
    )
)

selection_csv_hash_exact_628hc4f1b = bool(
    selection_csv_hash_reload_628hc4f1b
    ==
    policy_reload_628hc4f1b[
        "frozen_file_sha256"
    ][
        "policy_selection_audit_csv"
    ]
)

family_csv_hash_exact_628hc4f1b = bool(
    family_csv_hash_reload_628hc4f1b
    ==
    policy_reload_628hc4f1b[
        "frozen_file_sha256"
    ][
        "selected_family_severity_csv"
    ]
)

# ---------------------------------------------------------------
# 5. Verify upstream frozen-reliability lineage
# ---------------------------------------------------------------

upstream_dir_628hc4f1b = (
    project_root_628hc4f1b
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c3a3"
)

upstream_risk_path_628hc4f1b = (
    upstream_dir_628hc4f1b
    /
    policy_reload_628hc4f1b[
        "upstream_reliability_artifact"
    ][
        "filename"
    ]
)

if not upstream_risk_path_628hc4f1b.exists():
    raise FileNotFoundError(
        "Upstream frozen reliability artifact is missing."
    )

upstream_hash_reload_628hc4f1b = (
    sha256_file_628hc4f1b(
        upstream_risk_path_628hc4f1b
    )
)

upstream_hash_exact_628hc4f1b = bool(
    upstream_hash_reload_628hc4f1b
    ==
    policy_reload_628hc4f1b[
        "upstream_reliability_artifact"
    ][
        "sha256"
    ]
)

# ---------------------------------------------------------------
# 6. Metadata-contract verification
# ---------------------------------------------------------------

metadata_contract_exact_628hc4f1b = bool(

    policy_reload_628hc4f1b[
        "dataset"
    ]
    ==
    "NASA C-MAPSS FD002"

    and

    policy_reload_628hc4f1b[
        "step"
    ]
    ==
    "6.28H-C4-F1-A"

    and

    policy_reload_628hc4f1b[
        "selection_source_step"
    ]
    ==
    "6.28H-C4-A1"

    and

    policy_reload_628hc4f1b[
        "threshold_source_step"
    ]
    ==
    "6.28H-C4-A0"

    and

    policy_reload_628hc4f1b[
        "upstream_reliability_freeze_step"
    ]
    ==
    "6.28H-C3-A3-A3-F2-B"

    and

    policy_reload_628hc4f1b[
        "freeze_status"
    ]
    ==
    "WRITE_COMPLETE_RELOAD_NOT_YET_VERIFIED"

    and

    policy_reload_628hc4f1b[
        "reliability_architecture"
    ]
    ==
    "uncertainty_only"

    and

    policy_reload_628hc4f1b[
        "operating_policy"
    ]
    ==
    "uncertainty_only_fixed_threshold"

    and

    policy_reload_628hc4f1b[
        "acceptance_rule"
    ]
    ==
    "accept iff uncertainty_risk <= risk_threshold"

    and

    float(
        policy_reload_628hc4f1b[
            "selected_nominal_coverage"
        ]
    )
    ==
    0.80

    and

    np.isclose(
        float(
            policy_reload_628hc4f1b[
                "selected_nominal_abstention"
            ]
        ),
        0.20,
        rtol=0.0,
        atol=1e-15,
    )

    and

    policy_reload_628hc4f1b[
        "selection_contract"
    ][
        "minimum_nominal_coverage"
    ]
    ==
    0.80

    and

    policy_reload_628hc4f1b[
        "selection_contract"
    ][
        "maximum_q10_coverage_shortfall"
    ]
    ==
    0.02

    and

    policy_reload_628hc4f1b[
        "selection_contract"
    ][
        "objective"
    ]
    ==
    "maximize q10 scenario gain among eligible policies"

    and

    policy_reload_628hc4f1b[
        "selection_contract"
    ][
        "tie_break"
    ]
    ==
    "higher nominal coverage"

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "official_test_labels_read"
    ]
    is False

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "test_scored"
    ]
    is False

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "abstention_threshold_selected"
    ]
    is True

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "threshold_selected_before_test"
    ]
    is True

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "model_executed"
    ]
    is False

    and

    policy_reload_628hc4f1b[
        "methodological_status"
    ][
        "model_retrained"
    ]
    is False
)

# ---------------------------------------------------------------
# 7. Exact selected policy/threshold verification vs C4-A1 memory
# ---------------------------------------------------------------

selected_policy_reload_628hc4f1b = str(
    policy_reload_628hc4f1b[
        "operating_policy"
    ]
)

selected_coverage_reload_628hc4f1b = float(
    policy_reload_628hc4f1b[
        "selected_nominal_coverage"
    ]
)

selected_threshold_reload_628hc4f1b = float(
    policy_reload_628hc4f1b[
        "selected_exact_risk_threshold"
    ]
)

selected_policy_exact_628hc4f1b = bool(
    selected_policy_reload_628hc4f1b
    ==
    FD002_C4A1_SELECTED_POLICY_NAME_628HC4A1
)

selected_coverage_exact_628hc4f1b = bool(
    selected_coverage_reload_628hc4f1b
    ==
    float(
        FD002_C4A1_SELECTED_TARGET_COVERAGE_628HC4A1
    )
)

selected_threshold_exact_628hc4f1b = bool(
    selected_threshold_reload_628hc4f1b
    ==
    float(
        FD002_C4A1_SELECTED_RISK_THRESHOLD_628HC4A1
    )
)

# ---------------------------------------------------------------
# 8. Selection CSV semantic verification vs C4-A1 memory
# ---------------------------------------------------------------

selection_memory_628hc4f1b = (
    FD002_C4A1_POLICY_SELECTION_AUDIT_628HC4A1
    .copy()
    .sort_values(
        "target_coverage",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

selection_reload_628hc4f1b = (
    selection_reload_628hc4f1b
    .sort_values(
        "target_coverage",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

selection_columns_exact_628hc4f1b = bool(
    list(
        selection_reload_628hc4f1b.columns
    )
    ==
    list(
        selection_memory_628hc4f1b.columns
    )
)

selection_rows_exact_628hc4f1b = bool(
    len(
        selection_reload_628hc4f1b
    )
    ==
    len(
        selection_memory_628hc4f1b
    )
    ==
    3
)

selection_bool_columns_628hc4f1b = [
    "nominal_coverage_budget_ok",
    "clean_gain_positive",
    "q10_gain_positive",
    "worst_family_median_positive",
    "coverage_transfer_ok",
    "eligible",
]

selection_boolean_exact_628hc4f1b = bool(
    all(
        np.array_equal(
            selection_reload_628hc4f1b[
                column_628hc4f1b
            ]
            .astype(bool)
            .to_numpy(),

            selection_memory_628hc4f1b[
                column_628hc4f1b
            ]
            .astype(bool)
            .to_numpy(),
        )

        for column_628hc4f1b
        in selection_bool_columns_628hc4f1b
    )
)

selection_numeric_columns_628hc4f1b = [
    column_628hc4f1b
    for column_628hc4f1b
    in selection_memory_628hc4f1b.columns
    if column_628hc4f1b
    not in selection_bool_columns_628hc4f1b
]

selection_numeric_exact_628hc4f1b = bool(
    all(
        np.allclose(
            selection_reload_628hc4f1b[
                column_628hc4f1b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            selection_memory_628hc4f1b[
                column_628hc4f1b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            rtol=0.0,
            atol=1e-12,
        )

        for column_628hc4f1b
        in selection_numeric_columns_628hc4f1b
    )
)

selection_semantic_exact_628hc4f1b = bool(
    selection_columns_exact_628hc4f1b
    and
    selection_rows_exact_628hc4f1b
    and
    selection_boolean_exact_628hc4f1b
    and
    selection_numeric_exact_628hc4f1b
)

# ---------------------------------------------------------------
# 9. Family × severity CSV semantic verification
# ---------------------------------------------------------------

family_memory_628hc4f1b = (
    FD002_C4A1_SELECTED_FAMILY_SEVERITY_628HC4A1
    .copy()
    .reset_index(drop=True)
)

family_reload_628hc4f1b = (
    family_reload_628hc4f1b
    .reset_index(drop=True)
)

family_columns_exact_628hc4f1b = bool(
    list(
        family_reload_628hc4f1b.columns
    )
    ==
    list(
        family_memory_628hc4f1b.columns
    )
)

family_rows_exact_628hc4f1b = bool(
    len(
        family_reload_628hc4f1b
    )
    ==
    len(
        family_memory_628hc4f1b
    )
    ==
    15
)

family_text_exact_628hc4f1b = bool(
    np.array_equal(
        family_reload_628hc4f1b[
            [
                "family",
                "severity",
            ]
        ]
        .astype(str)
        .to_numpy(),

        family_memory_628hc4f1b[
            [
                "family",
                "severity",
            ]
        ]
        .astype(str)
        .to_numpy(),
    )
)

family_numeric_columns_628hc4f1b = [
    "target_coverage",
    "n_scenarios",
    "median_realized_coverage",
    "median_gain",
    "mean_gain",
]

family_numeric_exact_628hc4f1b = bool(
    all(
        np.allclose(
            family_reload_628hc4f1b[
                column_628hc4f1b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            family_memory_628hc4f1b[
                column_628hc4f1b
            ]
            .to_numpy(
                dtype=np.float64
            ),

            rtol=0.0,
            atol=1e-12,
        )

        for column_628hc4f1b
        in family_numeric_columns_628hc4f1b
    )
)

family_semantic_exact_628hc4f1b = bool(
    family_columns_exact_628hc4f1b
    and
    family_rows_exact_628hc4f1b
    and
    family_text_exact_628hc4f1b
    and
    family_numeric_exact_628hc4f1b
)

# ---------------------------------------------------------------
# 10. Verify frozen known-negative subgroup contract
# ---------------------------------------------------------------

negative_reload_628hc4f1b = (
    family_reload_628hc4f1b[
        family_reload_628hc4f1b[
            "median_gain"
        ]
        <
        0.0
    ]
    .copy()
    .reset_index(drop=True)
)

negative_pairs_reload_628hc4f1b = set(
    zip(
        negative_reload_628hc4f1b[
            "family"
        ].astype(str),

        negative_reload_628hc4f1b[
            "severity"
        ].astype(str),
    )
)

metadata_negative_pairs_628hc4f1b = set(
    (
        str(item_628hc4f1b["family"]),
        str(item_628hc4f1b["severity"]),
    )
    for item_628hc4f1b
    in policy_reload_628hc4f1b[
        "known_negative_family_severity_groups"
    ]
)

negative_subgroups_exact_628hc4f1b = bool(
    negative_pairs_reload_628hc4f1b
    ==
    {
        ("stuck", "medium"),
        ("stuck", "high"),
    }

    and

    metadata_negative_pairs_628hc4f1b
    ==
    negative_pairs_reload_628hc4f1b

    and

    len(
        negative_reload_628hc4f1b
    )
    ==
    2
)

# ---------------------------------------------------------------
# 11. Verify selected row remains eligible after reload
# ---------------------------------------------------------------

selected_row_reload_628hc4f1b = (
    selection_reload_628hc4f1b[
        np.isclose(
            selection_reload_628hc4f1b[
                "target_coverage"
            ].to_numpy(
                dtype=np.float64
            ),
            selected_coverage_reload_628hc4f1b,
            rtol=0.0,
            atol=1e-12,
        )
    ]
)

selected_row_unique_628hc4f1b = bool(
    len(
        selected_row_reload_628hc4f1b
    )
    ==
    1
)

selected_row_eligible_628hc4f1b = bool(
    selected_row_unique_628hc4f1b
    and
    bool(
        selected_row_reload_628hc4f1b
        .iloc[0][
            "eligible"
        ]
    )
)

selected_row_threshold_exact_628hc4f1b = bool(
    selected_row_unique_628hc4f1b
    and
    float(
        selected_row_reload_628hc4f1b
        .iloc[0][
            "risk_threshold"
        ]
    )
    ==
    selected_threshold_reload_628hc4f1b
)

# ---------------------------------------------------------------
# 12. Publish verified frozen policy objects in memory
# ---------------------------------------------------------------

FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B = (
    policy_reload_628hc4f1b
)

FD002_FROZEN_C4_POLICY_SELECTION_AUDIT_628HC4F1B = (
    selection_reload_628hc4f1b
)

FD002_FROZEN_C4_FAMILY_SEVERITY_628HC4F1B = (
    family_reload_628hc4f1b
)

FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B = (
    selected_policy_reload_628hc4f1b
)

FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B = (
    selected_coverage_reload_628hc4f1b
)

FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B = (
    selected_threshold_reload_628hc4f1b
)

# ---------------------------------------------------------------
# 13. Independent reload audit
# ---------------------------------------------------------------

audit_628hc4f1b = pd.DataFrame(
    {
        "check": [
            "all_three_F1A_files_present",
            "selection_csv_hash_exact",
            "family_csv_hash_exact",
            "upstream_reliability_hash_exact",
            "metadata_contract_exact",
            "selected_policy_exact_vs_A1",
            "selected_coverage_exact_vs_A1",
            "selected_threshold_exact_vs_A1",
            "selection_csv_semantic_exact",
            "family_csv_semantic_exact",
            "negative_subgroups_exact",
            "selected_row_unique",
            "selected_row_eligible",
            "selected_row_threshold_exact",
        ],

        "passed": [
            all(
                path_628hc4f1b.exists()
                for path_628hc4f1b
                in required_paths_628hc4f1b
            ),

            selection_csv_hash_exact_628hc4f1b,

            family_csv_hash_exact_628hc4f1b,

            upstream_hash_exact_628hc4f1b,

            metadata_contract_exact_628hc4f1b,

            selected_policy_exact_628hc4f1b,

            selected_coverage_exact_628hc4f1b,

            selected_threshold_exact_628hc4f1b,

            selection_semantic_exact_628hc4f1b,

            family_semantic_exact_628hc4f1b,

            negative_subgroups_exact_628hc4f1b,

            selected_row_unique_628hc4f1b,

            selected_row_eligible_628hc4f1b,

            selected_row_threshold_exact_628hc4f1b,
        ],
    }
)

print(
    "\n--- C4-F1-B INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc4f1b
)

all_checks_passed_628hc4f1b = bool(
    audit_628hc4f1b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc4f1b:
    raise RuntimeError(
        "C4-F1-B selected-policy verification failed."
    )

# ---------------------------------------------------------------
# 14. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C4-F1-B FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Reliability architecture        :",
    policy_reload_628hc4f1b[
        "reliability_architecture"
    ],
)

print(
    "Operating policy                :",
    selected_policy_reload_628hc4f1b,
)

print(
    "Selected nominal coverage       :",
    selected_coverage_reload_628hc4f1b,
)

print(
    "Selected exact risk threshold   :",
    repr(
        selected_threshold_reload_628hc4f1b
    ),
)

print(
    "Nominal abstention fraction     :",
    policy_reload_628hc4f1b[
        "selected_nominal_abstention"
    ],
)

print(
    "Selection CSV hash exact        :",
    selection_csv_hash_exact_628hc4f1b,
)

print(
    "Family CSV hash exact           :",
    family_csv_hash_exact_628hc4f1b,
)

print(
    "Upstream reliability hash exact :",
    upstream_hash_exact_628hc4f1b,
)

print(
    "Selection table exact           :",
    selection_semantic_exact_628hc4f1b,
)

print(
    "Family×severity table exact     :",
    family_semantic_exact_628hc4f1b,
)

print(
    "Known negative subgroups exact  :",
    negative_subgroups_exact_628hc4f1b,
)

print(
    "Independent reload verified     :",
    all_checks_passed_628hc4f1b,
)

print("DEV RUL labels used in this step : False")
print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Abstention threshold selected    : True")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files modified                   : False")

print(
    "\n=== END STEP 6.28H-C4-F1-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-A0
# UNTOUCHED FD002 TEST INFERENCE-READINESS INVENTORY
# ================================================================

from pathlib import Path
import inspect
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A0 — UNTOUCHED FD002 TEST INFERENCE-READINESS INVENTORY")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Verify the frozen C4 policy contract
# ---------------------------------------------------------------

frozen_policy_metadata_628hc5a0 = (
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B
)

frozen_policy_name_628hc5a0 = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

frozen_coverage_628hc5a0 = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

frozen_threshold_628hc5a0 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

freeze_contract_ok_628hc5a0 = bool(

    frozen_policy_metadata_628hc5a0[
        "reliability_architecture"
    ]
    ==
    "uncertainty_only"

    and

    frozen_policy_name_628hc5a0
    ==
    "uncertainty_only_fixed_threshold"

    and

    frozen_coverage_628hc5a0
    ==
    0.80

    and

    frozen_threshold_628hc5a0
    ==
    0.8003345892330226
)

if not freeze_contract_ok_628hc5a0:
    raise RuntimeError(
        "Frozen C4 policy contract does not match the verified F1-B state."
    )

print("\n--- FROZEN PRE-TEST CONTRACT ---")

print(
    "Reliability architecture :",
    frozen_policy_metadata_628hc5a0[
        "reliability_architecture"
    ],
)

print(
    "Operating policy         :",
    frozen_policy_name_628hc5a0,
)

print(
    "Nominal coverage         :",
    frozen_coverage_628hc5a0,
)

print(
    "Exact risk threshold     :",
    repr(
        frozen_threshold_628hc5a0
    ),
)

# ---------------------------------------------------------------
# 2. Resolve project root
# ---------------------------------------------------------------

cwd_628hc5a0 = Path.cwd()

if (cwd_628hc5a0 / "06_results").exists():

    project_root_628hc5a0 = (
        cwd_628hc5a0
    )

elif (
    cwd_628hc5a0.parent
    /
    "06_results"
).exists():

    project_root_628hc5a0 = (
        cwd_628hc5a0.parent
    )

else:

    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )

print(
    "\nProject root:",
    project_root_628hc5a0,
)

# ---------------------------------------------------------------
# 3. Locate official FD002 raw TEST files by filename ONLY
#
# IMPORTANT:
#     RUL_FD002.txt is NOT opened or read here.
# ---------------------------------------------------------------

test_feature_paths_628hc5a0 = sorted(
    project_root_628hc5a0.rglob(
        "test_FD002.txt"
    )
)

test_label_paths_628hc5a0 = sorted(
    project_root_628hc5a0.rglob(
        "RUL_FD002.txt"
    )
)

print(
    "\n--- OFFICIAL FD002 FILE-PATH INVENTORY ---"
)

print(
    "test_FD002.txt matches :",
    len(
        test_feature_paths_628hc5a0
    ),
)

for path_628hc5a0 in (
    test_feature_paths_628hc5a0
):
    print(
        "  TEST features:",
        path_628hc5a0,
    )

print(
    "RUL_FD002.txt matches  :",
    len(
        test_label_paths_628hc5a0
    ),
)

for path_628hc5a0 in (
    test_label_paths_628hc5a0
):
    print(
        "  TEST label path only:",
        path_628hc5a0,
    )

# ---------------------------------------------------------------
# 4. Snapshot globals safely
#
# list(...) prevents the old
# "dictionary changed size during iteration" problem.
# ---------------------------------------------------------------

globals_snapshot_628hc5a0 = list(
    globals().items()
)

# ---------------------------------------------------------------
# 5. Inventory existing TEST-related in-memory objects
#
# Metadata only:
#     name
#     Python type
#     shape
#
# No contents are inspected.
# ---------------------------------------------------------------

test_object_rows_628hc5a0 = []

for (
    name_628hc5a0,
    obj_628hc5a0,
) in globals_snapshot_628hc5a0:

    upper_name_628hc5a0 = (
        str(
            name_628hc5a0
        )
        .upper()
    )

    if (
        "TEST"
        not in
        upper_name_628hc5a0
    ):
        continue

    if str(
        name_628hc5a0
    ).endswith(
        "_628hc5a0"
    ):
        continue

    shape_628hc5a0 = getattr(
        obj_628hc5a0,
        "shape",
        None,
    )

    test_object_rows_628hc5a0.append(
        {
            "name":
                str(
                    name_628hc5a0
                ),

            "type":
                type(
                    obj_628hc5a0
                ).__name__,

            "shape":
                (
                    str(
                        tuple(
                            shape_628hc5a0
                        )
                    )
                    if shape_628hc5a0 is not None
                    else ""
                ),
        }
    )

FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0 = (
    pd.DataFrame(
        test_object_rows_628hc5a0
    )
)

if len(
    FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0
) > 0:

    FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0 = (
        FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0
        .sort_values(
            "name",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

print(
    "\n--- EXISTING TEST-RELATED IN-MEMORY OBJECTS ---"
)

display(
    FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0
)

# ---------------------------------------------------------------
# 6. Inventory prediction-capable model-like objects
#
# Nothing is executed.
# ---------------------------------------------------------------

model_rows_628hc5a0 = []

for (
    name_628hc5a0,
    obj_628hc5a0,
) in globals_snapshot_628hc5a0:

    if str(
        name_628hc5a0
    ).endswith(
        "_628hc5a0"
    ):
        continue

    has_predict_628hc5a0 = bool(
        hasattr(
            obj_628hc5a0,
            "predict",
        )
        and
        callable(
            getattr(
                obj_628hc5a0,
                "predict",
            )
        )
    )

    if not has_predict_628hc5a0:
        continue

    model_rows_628hc5a0.append(
        {
            "name":
                str(
                    name_628hc5a0
                ),

            "type":
                type(
                    obj_628hc5a0
                ).__name__,

            "module":
                str(
                    type(
                        obj_628hc5a0
                    ).__module__
                ),
        }
    )

FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0 = (
    pd.DataFrame(
        model_rows_628hc5a0
    )
)

if len(
    FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0
) > 0:

    FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0 = (
        FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0
        .sort_values(
            "name",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

print(
    "\n--- PREDICTION-CAPABLE MODEL OBJECTS ---"
)

display(
    FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0
)

# ---------------------------------------------------------------
# 7. Inventory relevant frozen pipeline helper functions
#
# Functions are NOT executed.
# ---------------------------------------------------------------

helper_keywords_628hc5a0 = (
    "NORMAL",
    "WINDOW",
    "PREDICT",
    "DROPOUT",
    "MC",
    "SENSOR",
)

helper_rows_628hc5a0 = []

for (
    name_628hc5a0,
    obj_628hc5a0,
) in globals_snapshot_628hc5a0:

    if not inspect.isfunction(
        obj_628hc5a0
    ):
        continue

    upper_name_628hc5a0 = (
        str(
            name_628hc5a0
        )
        .upper()
    )

    if not any(
        keyword_628hc5a0
        in
        upper_name_628hc5a0

        for keyword_628hc5a0
        in helper_keywords_628hc5a0
    ):
        continue

    try:

        signature_628hc5a0 = str(
            inspect.signature(
                obj_628hc5a0
            )
        )

    except (
        TypeError,
        ValueError,
    ):

        signature_628hc5a0 = (
            "<signature unavailable>"
        )

    helper_rows_628hc5a0.append(
        {
            "name":
                str(
                    name_628hc5a0
                ),

            "signature":
                signature_628hc5a0,
        }
    )

FD002_C5A0_HELPER_FUNCTION_INVENTORY_628HC5A0 = (
    pd.DataFrame(
        helper_rows_628hc5a0
    )
)

if len(
    FD002_C5A0_HELPER_FUNCTION_INVENTORY_628HC5A0
) > 0:

    FD002_C5A0_HELPER_FUNCTION_INVENTORY_628HC5A0 = (
        FD002_C5A0_HELPER_FUNCTION_INVENTORY_628HC5A0
        .sort_values(
            "name",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

print(
    "\n--- RELEVANT PIPELINE HELPER FUNCTIONS ---"
)

display(
    FD002_C5A0_HELPER_FUNCTION_INVENTORY_628HC5A0
)

# ---------------------------------------------------------------
# 8. Check for suspicious already-loaded TEST-label arrays
#
# This is a NAME/TYPE/SHAPE audit only.
# It does not read any label file.
# ---------------------------------------------------------------

label_name_tokens_628hc5a0 = (
    "Y_TEST",
    "TEST_RUL",
    "RUL_TEST",
    "RUL_FD002",
)

possible_loaded_label_rows_628hc5a0 = []

for (
    name_628hc5a0,
    obj_628hc5a0,
) in globals_snapshot_628hc5a0:

    upper_name_628hc5a0 = (
        str(
            name_628hc5a0
        )
        .upper()
    )

    if not any(
        token_628hc5a0
        in
        upper_name_628hc5a0

        for token_628hc5a0
        in label_name_tokens_628hc5a0
    ):
        continue

    if isinstance(
        obj_628hc5a0,
        (
            str,
            Path,
        ),
    ):
        continue

    shape_628hc5a0 = getattr(
        obj_628hc5a0,
        "shape",
        None,
    )

    possible_loaded_label_rows_628hc5a0.append(
        {
            "name":
                str(
                    name_628hc5a0
                ),

            "type":
                type(
                    obj_628hc5a0
                ).__name__,

            "shape":
                (
                    str(
                        tuple(
                            shape_628hc5a0
                        )
                    )
                    if shape_628hc5a0 is not None
                    else ""
                ),
        }
    )

FD002_C5A0_POSSIBLE_TEST_LABEL_OBJECTS_628HC5A0 = (
    pd.DataFrame(
        possible_loaded_label_rows_628hc5a0
    )
)

print(
    "\n--- POSSIBLE ALREADY-LOADED TEST-LABEL OBJECTS ---"
)

display(
    FD002_C5A0_POSSIBLE_TEST_LABEL_OBJECTS_628HC5A0
)

# ---------------------------------------------------------------
# 9. C5-A0 audit
# ---------------------------------------------------------------

audit_628hc5a0 = pd.DataFrame(
    {
        "check": [
            "frozen_policy_contract_exact",
            "frozen_architecture_uncertainty_only",
            "frozen_nominal_coverage_080",
            "frozen_threshold_exact",
            "test_feature_filename_inventory_completed",
            "test_label_filename_inventory_completed_without_read",
            "globals_snapshot_safe",
            "no_model_execution",
            "no_test_scoring",
            "no_files_written",
        ],

        "passed": [
            freeze_contract_ok_628hc5a0,

            frozen_policy_metadata_628hc5a0[
                "reliability_architecture"
            ]
            ==
            "uncertainty_only",

            frozen_coverage_628hc5a0
            ==
            0.80,

            frozen_threshold_628hc5a0
            ==
            0.8003345892330226,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)

print(
    "\n--- C5-A0 AUDIT ---"
)

display(
    audit_628hc5a0
)

all_checks_passed_628hc5a0 = bool(
    audit_628hc5a0[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a0:

    raise RuntimeError(
        "C5-A0 TEST-readiness inventory audit failed."
    )

# ---------------------------------------------------------------
# 10. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C5-A0 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Frozen reliability architecture    : uncertainty_only"
)

print(
    "Frozen operating policy            :",
    frozen_policy_name_628hc5a0,
)

print(
    "Frozen nominal coverage            :",
    frozen_coverage_628hc5a0,
)

print(
    "Frozen exact threshold             :",
    repr(
        frozen_threshold_628hc5a0
    ),
)

print(
    "test_FD002.txt path matches        :",
    len(
        test_feature_paths_628hc5a0
    ),
)

print(
    "RUL_FD002.txt path matches         :",
    len(
        test_label_paths_628hc5a0
    ),
)

print(
    "Existing TEST-related objects      :",
    len(
        FD002_C5A0_TEST_OBJECT_INVENTORY_628HC5A0
    ),
)

print(
    "Prediction-capable model objects   :",
    len(
        FD002_C5A0_MODEL_OBJECT_INVENTORY_628HC5A0
    ),
)

print(
    "Possible loaded TEST-label objects :",
    len(
        FD002_C5A0_POSSIBLE_TEST_LABEL_OBJECTS_628HC5A0
    ),
)

print(
    "All checks passed                  :",
    all_checks_passed_628hc5a0,
)

print("Official TEST feature file read     : False")
print("Official TEST labels read           : False")
print("TEST scored                         : False")
print("Frozen threshold modified           : False")
print("Model executed                      : False")
print("Model retrained                     : False")
print("Files written                       : False")

print(
    "\n=== END STEP 6.28H-C5-A0 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-A0-R1
# PRE-EXISTING FD002 TEST ARTIFACT LINEAGE CHECK
# ================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A0-R1 — PRE-EXISTING FD002 TEST ARTIFACT LINEAGE CHECK")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Existing status markers: print their ACTUAL values
# ---------------------------------------------------------------

marker_names_628hc5a0r1 = [
    "FD002_OFFICIAL_TEST_LABELS_READ_628A",
    "FD002_PRIMARY_TEST_POPULATION_628D",
    "EXPECTED_TEST_628FB",
]

marker_rows_628hc5a0r1 = []

for name in marker_names_628hc5a0r1:
    if name in globals():
        obj = globals()[name]
        marker_rows_628hc5a0r1.append({
            "name": name,
            "exists": True,
            "type": type(obj).__name__,
            "value": repr(obj) if np.isscalar(obj) else "<non-scalar>",
        })
    else:
        marker_rows_628hc5a0r1.append({
            "name": name,
            "exists": False,
            "type": "<missing>",
            "value": "<missing>",
        })

marker_table_628hc5a0r1 = pd.DataFrame(marker_rows_628hc5a0r1)

print("\n--- ACTUAL PRE-EXISTING TEST STATUS MARKERS ---")
display(marker_table_628hc5a0r1)

# ---------------------------------------------------------------
# 2. Key TEST artifacts already present in memory
# ---------------------------------------------------------------

key_names_628hc5a0r1 = [
    "test_raw_628c",
    "test_regime_628c",
    "test_u",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
]

artifact_rows_628hc5a0r1 = []

for name in key_names_628hc5a0r1:
    if name in globals():
        obj = globals()[name]
        artifact_rows_628hc5a0r1.append({
            "name": name,
            "exists": True,
            "type": type(obj).__name__,
            "shape": str(getattr(obj, "shape", "")),
            "dtype": str(getattr(obj, "dtype", "")),
        })
    else:
        artifact_rows_628hc5a0r1.append({
            "name": name,
            "exists": False,
            "type": "<missing>",
            "shape": "",
            "dtype": "",
        })

artifact_table_628hc5a0r1 = pd.DataFrame(artifact_rows_628hc5a0r1)

print("\n--- KEY PRE-EXISTING TEST ARTIFACTS ---")
display(artifact_table_628hc5a0r1)

# ---------------------------------------------------------------
# 3. Compact inventory of TEST-related endpoint vectors
#    Official FD002 TEST population = 259 engines.
#    Values are NOT printed.
# ---------------------------------------------------------------

endpoint_rows_628hc5a0r1 = []

for name, obj in list(globals().items()):
    shape = getattr(obj, "shape", None)

    if shape is None or tuple(shape) != (259,):
        continue

    upper = str(name).upper()

    if not any(
        token in upper
        for token in ("TEST", "PRED", "UNCERTAINT", "RISK")
    ):
        continue

    endpoint_rows_628hc5a0r1.append({
        "name": str(name),
        "type": type(obj).__name__,
        "shape": str(tuple(shape)),
        "dtype": str(getattr(obj, "dtype", "")),
        "label_like_name": any(
            token in upper
            for token in ("LABEL", "RUL", "TARGET", "TRUTH", "Y_TEST")
        ),
    })

endpoint_table_628hc5a0r1 = (
    pd.DataFrame(endpoint_rows_628hc5a0r1)
    .sort_values("name", kind="mergesort")
    .reset_index(drop=True)
    if endpoint_rows_628hc5a0r1
    else pd.DataFrame(
        columns=["name", "type", "shape", "dtype", "label_like_name"]
    )
)

print("\n--- TEST-RELATED ENDPOINT VECTORS OF LENGTH 259 ---")
display(endpoint_table_628hc5a0r1)

# ---------------------------------------------------------------
# 4. Recover source for specific old TEST-lineage helpers
#    Nothing is executed.
# ---------------------------------------------------------------

helper_names_628hc5a0r1 = [
    "build_test_endpoint_windows_628d",
    "test_prediction_candidates_628fb",
    "mc_std_candidates_628fb",
    "mc_dropout_predict_628fbr1",
]

print("\n--- OLD TEST-LINEAGE HELPER SOURCE ---")

source_recovered_628hc5a0r1 = True

for name in helper_names_628hc5a0r1:

    print("\n" + "-" * 94)
    print(name)
    print("-" * 94)

    if name not in globals():
        print("<missing>")
        source_recovered_628hc5a0r1 = False
        continue

    func = globals()[name]

    try:
        print("Signature:", inspect.signature(func))
        print(inspect.getsource(func))

    except (OSError, TypeError, ValueError) as exc:
        print("Source unavailable:", repr(exc))
        source_recovered_628hc5a0r1 = False

# ---------------------------------------------------------------
# 5. Conservative lineage summary
# ---------------------------------------------------------------

marker_exists_628hc5a0r1 = (
    "FD002_OFFICIAL_TEST_LABELS_READ_628A" in globals()
)

marker_value_628hc5a0r1 = (
    bool(FD002_OFFICIAL_TEST_LABELS_READ_628A)
    if marker_exists_628hc5a0r1
    else None
)

preexisting_test_features_628hc5a0r1 = (
    "test_raw_628c" in globals()
)

preexisting_test_outputs_628hc5a0r1 = (
    len(endpoint_table_628hc5a0r1) > 0
)

label_like_vectors_628hc5a0r1 = (
    int(endpoint_table_628hc5a0r1["label_like_name"].sum())
    if len(endpoint_table_628hc5a0r1)
    else 0
)

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A0-R1 FINAL SUMMARY")
print("=" * 94)

print(
    "Pre-existing TEST feature object present :",
    preexisting_test_features_628hc5a0r1,
)

print(
    "Pre-existing TEST endpoint outputs       :",
    preexisting_test_outputs_628hc5a0r1,
)

print(
    "Official TEST-label marker exists        :",
    marker_exists_628hc5a0r1,
)

print(
    "Official TEST-label marker value         :",
    marker_value_628hc5a0r1,
)

print(
    "Label-like endpoint vectors found        :",
    label_like_vectors_628hc5a0r1,
)

print(
    "Requested helper source recovered        :",
    source_recovered_628hc5a0r1,
)

print("TEST files read in THIS step              : False")
print("TEST scored in THIS step                  : False")
print("Frozen C4 policy modified                 : False")
print("Model executed in THIS step               : False")
print("Model retrained                           : False")
print("Files written                             : False")

print("\n=== END STEP 6.28H-C5-A0-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A0-R1-R1
# PRE-EXISTING FD002 TEST ARTIFACT LINEAGE CHECK — SAFE REPAIR
# ================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A0-R1-R1 — PRE-EXISTING FD002 TEST ARTIFACT LINEAGE CHECK")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Existing status markers: print their ACTUAL values
# ---------------------------------------------------------------

marker_names_628hc5a0r1r1 = [
    "FD002_OFFICIAL_TEST_LABELS_READ_628A",
    "FD002_PRIMARY_TEST_POPULATION_628D",
    "EXPECTED_TEST_628FB",
]

marker_rows_628hc5a0r1r1 = []

for name_628hc5a0r1r1 in marker_names_628hc5a0r1r1:

    if name_628hc5a0r1r1 in globals():

        obj_628hc5a0r1r1 = globals()[name_628hc5a0r1r1]

        marker_rows_628hc5a0r1r1.append(
            {
                "name": name_628hc5a0r1r1,
                "exists": True,
                "type": type(obj_628hc5a0r1r1).__name__,
                "value": (
                    repr(obj_628hc5a0r1r1)
                    if np.isscalar(obj_628hc5a0r1r1)
                    else "<non-scalar>"
                ),
            }
        )

    else:

        marker_rows_628hc5a0r1r1.append(
            {
                "name": name_628hc5a0r1r1,
                "exists": False,
                "type": "<missing>",
                "value": "<missing>",
            }
        )

marker_table_628hc5a0r1r1 = pd.DataFrame(
    marker_rows_628hc5a0r1r1
)

print("\n--- ACTUAL PRE-EXISTING TEST STATUS MARKERS ---")
display(marker_table_628hc5a0r1r1)

# ---------------------------------------------------------------
# 2. Key TEST artifacts already present in memory
# ---------------------------------------------------------------

key_names_628hc5a0r1r1 = [
    "test_raw_628c",
    "test_regime_628c",
    "test_u",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
]

artifact_rows_628hc5a0r1r1 = []

for name_628hc5a0r1r1 in key_names_628hc5a0r1r1:

    if name_628hc5a0r1r1 in globals():

        obj_628hc5a0r1r1 = globals()[name_628hc5a0r1r1]

        artifact_rows_628hc5a0r1r1.append(
            {
                "name": name_628hc5a0r1r1,
                "exists": True,
                "type": type(obj_628hc5a0r1r1).__name__,
                "shape": str(
                    getattr(
                        obj_628hc5a0r1r1,
                        "shape",
                        "",
                    )
                ),
                "dtype": str(
                    getattr(
                        obj_628hc5a0r1r1,
                        "dtype",
                        "",
                    )
                ),
            }
        )

    else:

        artifact_rows_628hc5a0r1r1.append(
            {
                "name": name_628hc5a0r1r1,
                "exists": False,
                "type": "<missing>",
                "shape": "",
                "dtype": "",
            }
        )

artifact_table_628hc5a0r1r1 = pd.DataFrame(
    artifact_rows_628hc5a0r1r1
)

print("\n--- KEY PRE-EXISTING TEST ARTIFACTS ---")
display(artifact_table_628hc5a0r1r1)

# ---------------------------------------------------------------
# 3. Safe snapshot of globals
# ---------------------------------------------------------------

globals_snapshot_628hc5a0r1r1 = list(
    globals().items()
)

# ---------------------------------------------------------------
# 4. Compact inventory of TEST-related endpoint vectors
#
# IMPORTANT REPAIR:
# inspect only concrete vector containers.
# No tuple(obj.shape) on arbitrary objects.
# ---------------------------------------------------------------

endpoint_rows_628hc5a0r1r1 = []

for (
    name_628hc5a0r1r1,
    obj_628hc5a0r1r1,
) in globals_snapshot_628hc5a0r1r1:

    name_text_628hc5a0r1r1 = str(
        name_628hc5a0r1r1
    )

    upper_name_628hc5a0r1r1 = (
        name_text_628hc5a0r1r1.upper()
    )

    if not any(
        token_628hc5a0r1r1
        in upper_name_628hc5a0r1r1

        for token_628hc5a0r1r1
        in (
            "TEST",
            "PRED",
            "UNCERTAINT",
            "RISK",
        )
    ):
        continue

    vector_length_628hc5a0r1r1 = None
    shape_text_628hc5a0r1r1 = ""
    dtype_text_628hc5a0r1r1 = ""

    if isinstance(
        obj_628hc5a0r1r1,
        np.ndarray,
    ):

        if obj_628hc5a0r1r1.ndim != 1:
            continue

        vector_length_628hc5a0r1r1 = int(
            obj_628hc5a0r1r1.shape[0]
        )

        shape_text_628hc5a0r1r1 = str(
            obj_628hc5a0r1r1.shape
        )

        dtype_text_628hc5a0r1r1 = str(
            obj_628hc5a0r1r1.dtype
        )

    elif isinstance(
        obj_628hc5a0r1r1,
        pd.Series,
    ):

        vector_length_628hc5a0r1r1 = int(
            len(obj_628hc5a0r1r1)
        )

        shape_text_628hc5a0r1r1 = str(
            obj_628hc5a0r1r1.shape
        )

        dtype_text_628hc5a0r1r1 = str(
            obj_628hc5a0r1r1.dtype
        )

    elif isinstance(
        obj_628hc5a0r1r1,
        (
            list,
            tuple,
        ),
    ):

        vector_length_628hc5a0r1r1 = int(
            len(obj_628hc5a0r1r1)
        )

        shape_text_628hc5a0r1r1 = (
            f"({vector_length_628hc5a0r1r1},)"
        )

        dtype_text_628hc5a0r1r1 = (
            type(obj_628hc5a0r1r1).__name__
        )

    else:
        continue

    if vector_length_628hc5a0r1r1 != 259:
        continue

    endpoint_rows_628hc5a0r1r1.append(
        {
            "name": name_text_628hc5a0r1r1,
            "type": type(obj_628hc5a0r1r1).__name__,
            "shape": shape_text_628hc5a0r1r1,
            "dtype": dtype_text_628hc5a0r1r1,
            "label_like_name": any(
                token_628hc5a0r1r1
                in upper_name_628hc5a0r1r1

                for token_628hc5a0r1r1
                in (
                    "LABEL",
                    "RUL",
                    "TARGET",
                    "TRUTH",
                    "Y_TEST",
                )
            ),
        }
    )

if endpoint_rows_628hc5a0r1r1:

    endpoint_table_628hc5a0r1r1 = (
        pd.DataFrame(
            endpoint_rows_628hc5a0r1r1
        )
        .sort_values(
            "name",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

else:

    endpoint_table_628hc5a0r1r1 = pd.DataFrame(
        columns=[
            "name",
            "type",
            "shape",
            "dtype",
            "label_like_name",
        ]
    )

print("\n--- TEST-RELATED ENDPOINT VECTORS OF LENGTH 259 ---")
display(endpoint_table_628hc5a0r1r1)

# ---------------------------------------------------------------
# 5. Recover source for specific old TEST-lineage helpers
#
# Nothing is executed.
# ---------------------------------------------------------------

helper_names_628hc5a0r1r1 = [
    "build_test_endpoint_windows_628d",
    "test_prediction_candidates_628fb",
    "mc_std_candidates_628fb",
    "deterministic_predict_628fbr1",
    "mc_dropout_predict_628fbr1",
]

helper_source_rows_628hc5a0r1r1 = []

print("\n--- OLD TEST-LINEAGE HELPER SOURCE ---")

for name_628hc5a0r1r1 in helper_names_628hc5a0r1r1:

    print("\n" + "-" * 94)
    print(name_628hc5a0r1r1)
    print("-" * 94)

    if name_628hc5a0r1r1 not in globals():

        print("<missing>")

        helper_source_rows_628hc5a0r1r1.append(
            {
                "name": name_628hc5a0r1r1,
                "exists": False,
                "signature": "<missing>",
                "source_recovered": False,
            }
        )

        continue

    func_628hc5a0r1r1 = globals()[
        name_628hc5a0r1r1
    ]

    try:

        signature_628hc5a0r1r1 = str(
            inspect.signature(
                func_628hc5a0r1r1
            )
        )

    except (
        TypeError,
        ValueError,
    ):

        signature_628hc5a0r1r1 = (
            "<signature unavailable>"
        )

    print(
        "Signature:",
        signature_628hc5a0r1r1,
    )

    try:

        source_628hc5a0r1r1 = (
            inspect.getsource(
                func_628hc5a0r1r1
            )
        )

        print(
            source_628hc5a0r1r1
        )

        source_recovered_one_628hc5a0r1r1 = True

    except (
        OSError,
        TypeError,
    ) as exc_628hc5a0r1r1:

        print(
            "Source unavailable:",
            repr(exc_628hc5a0r1r1),
        )

        source_recovered_one_628hc5a0r1r1 = False

    helper_source_rows_628hc5a0r1r1.append(
        {
            "name": name_628hc5a0r1r1,
            "exists": True,
            "signature": signature_628hc5a0r1r1,
            "source_recovered": source_recovered_one_628hc5a0r1r1,
        }
    )

helper_source_table_628hc5a0r1r1 = pd.DataFrame(
    helper_source_rows_628hc5a0r1r1
)

print("\n--- HELPER SOURCE RECOVERY AUDIT ---")
display(helper_source_table_628hc5a0r1r1)

# ---------------------------------------------------------------
# 6. Conservative lineage checks
# ---------------------------------------------------------------

marker_exists_628hc5a0r1r1 = bool(
    "FD002_OFFICIAL_TEST_LABELS_READ_628A"
    in globals()
)

marker_value_628hc5a0r1r1 = (
    bool(FD002_OFFICIAL_TEST_LABELS_READ_628A)
    if marker_exists_628hc5a0r1r1
    else None
)

preexisting_test_features_628hc5a0r1r1 = bool(
    "test_raw_628c"
    in globals()
)

preexisting_test_outputs_628hc5a0r1r1 = bool(
    len(endpoint_table_628hc5a0r1r1) > 0
)

label_like_vectors_628hc5a0r1r1 = (
    int(
        endpoint_table_628hc5a0r1r1[
            "label_like_name"
        ].sum()
    )
    if len(endpoint_table_628hc5a0r1r1)
    else 0
)

all_requested_helpers_exist_628hc5a0r1r1 = bool(
    helper_source_table_628hc5a0r1r1[
        "exists"
    ].all()
)

all_requested_sources_recovered_628hc5a0r1r1 = bool(
    helper_source_table_628hc5a0r1r1[
        "source_recovered"
    ].all()
)

# ---------------------------------------------------------------
# 7. Audit
# ---------------------------------------------------------------

audit_628hc5a0r1r1 = pd.DataFrame(
    {
        "check": [
            "official_test_label_marker_exists",
            "official_test_label_marker_false",
            "primary_test_population_all_259_engines",
            "expected_test_population_259",
            "preexisting_test_feature_object_present",
            "preexisting_test_endpoint_outputs_present",
            "no_label_like_259_endpoint_vectors",
            "all_requested_helpers_exist",
            "all_requested_helper_sources_recovered",
            "no_test_file_read_in_this_step",
            "no_model_execution",
            "no_test_scoring",
            "no_files_written",
        ],

        "passed": [
            marker_exists_628hc5a0r1r1,

            (
                marker_exists_628hc5a0r1r1
                and
                marker_value_628hc5a0r1r1 is False
            ),

            (
                "FD002_PRIMARY_TEST_POPULATION_628D"
                in globals()
                and
                str(FD002_PRIMARY_TEST_POPULATION_628D)
                ==
                "all_259_engines"
            ),

            (
                "EXPECTED_TEST_628FB"
                in globals()
                and
                int(EXPECTED_TEST_628FB)
                ==
                259
            ),

            preexisting_test_features_628hc5a0r1r1,

            preexisting_test_outputs_628hc5a0r1r1,

            label_like_vectors_628hc5a0r1r1 == 0,

            all_requested_helpers_exist_628hc5a0r1r1,

            all_requested_sources_recovered_628hc5a0r1r1,

            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A0-R1-R1 AUDIT ---")
display(audit_628hc5a0r1r1)

all_checks_passed_628hc5a0r1r1 = bool(
    audit_628hc5a0r1r1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a0r1r1:

    raise RuntimeError(
        "C5-A0-R1-R1 TEST-artifact lineage audit did not fully pass."
    )

# ---------------------------------------------------------------
# 8. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A0-R1-R1 FINAL SUMMARY")
print("=" * 94)

print(
    "Pre-existing TEST feature object present :",
    preexisting_test_features_628hc5a0r1r1,
)

print(
    "Pre-existing TEST endpoint outputs       :",
    preexisting_test_outputs_628hc5a0r1r1,
)

print(
    "Official TEST-label marker exists        :",
    marker_exists_628hc5a0r1r1,
)

print(
    "Official TEST-label marker value         :",
    marker_value_628hc5a0r1r1,
)

print(
    "Label-like endpoint vectors found        :",
    label_like_vectors_628hc5a0r1r1,
)

print(
    "Requested helpers all exist              :",
    all_requested_helpers_exist_628hc5a0r1r1,
)

print(
    "Requested helper source recovered        :",
    all_requested_sources_recovered_628hc5a0r1r1,
)

print(
    "All checks passed                        :",
    all_checks_passed_628hc5a0r1r1,
)

print("TEST files read in THIS step              : False")
print("TEST scored in THIS step                  : False")
print("Frozen C4 policy modified                 : False")
print("Model executed in THIS step               : False")
print("Model retrained                           : False")
print("Files written                             : False")

print("\n=== END STEP 6.28H-C5-A0-R1-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A1
# EXISTING FD002 TEST UNCERTAINTY-RISK LINEAGE RECONCILIATION
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A1 — EXISTING FD002 TEST UNCERTAINTY-RISK LINEAGE RECONCILIATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconfirm frozen pre-test policy
# ---------------------------------------------------------------

frozen_architecture_628hc5a1 = str(
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B[
        "reliability_architecture"
    ]
)

frozen_policy_628hc5a1 = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

frozen_coverage_628hc5a1 = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

frozen_threshold_628hc5a1 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

if frozen_architecture_628hc5a1 != "uncertainty_only":
    raise RuntimeError(
        "Frozen reliability architecture changed unexpectedly."
    )

if frozen_policy_628hc5a1 != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Frozen operating policy changed unexpectedly."
    )

if frozen_coverage_628hc5a1 != 0.80:
    raise RuntimeError(
        "Frozen nominal coverage changed unexpectedly."
    )

if frozen_threshold_628hc5a1 != 0.8003345892330226:
    raise RuntimeError(
        "Frozen abstention threshold changed unexpectedly."
    )

# ---------------------------------------------------------------
# 2. Official TEST-label safety marker must still be false
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Historical TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is True. Stop before continuing."
    )

# ---------------------------------------------------------------
# 3. Explicit candidate objects
#
# No generic globals scan.
# Only known historical TEST uncertainty/risk objects are checked.
# ---------------------------------------------------------------

candidate_names_628hc5a1 = [
    "test_u",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",
    "FD002_TEST_SELECTIVE_RISK_628FC",
    "FD002_TEST_SELECTIVE_RISK_628FCR2",
    "stored_test_adjusted_628hc0r2",
    "reproduced_test_adjusted_628hc0r2",
    "stored_test_g1_628hc0r2",
    "reproduced_test_g1_628hc0r2",
    "stored_test_final_risk_628hc0r2",
    "reproduced_test_final_risk_628hc0r2",
    "test_adjusted_628fb",
    "test_adjusted_628fc",
    "test_adjusted_r1",
    "test_g1",
    "test_g1_628fc",
    "test_g1_628fcr2",
    "test_final",
]

candidate_arrays_628hc5a1 = {}
candidate_rows_628hc5a1 = []

for name_628hc5a1 in candidate_names_628hc5a1:

    if name_628hc5a1 not in globals():

        candidate_rows_628hc5a1.append(
            {
                "name": name_628hc5a1,
                "exists": False,
                "shape": "",
                "finite": False,
                "min": np.nan,
                "median": np.nan,
                "mean": np.nan,
                "max": np.nan,
                "within_0_1": False,
            }
        )

        continue

    array_628hc5a1 = np.asarray(
        globals()[name_628hc5a1],
        dtype=np.float64,
    ).reshape(-1)

    if array_628hc5a1.shape != (259,):
        raise RuntimeError(
            f"{name_628hc5a1} has unexpected shape "
            f"{array_628hc5a1.shape}."
        )

    finite_628hc5a1 = bool(
        np.isfinite(
            array_628hc5a1
        ).all()
    )

    within_0_1_628hc5a1 = bool(
        finite_628hc5a1
        and
        np.min(array_628hc5a1) >= 0.0
        and
        np.max(array_628hc5a1) <= 1.0
    )

    candidate_arrays_628hc5a1[
        name_628hc5a1
    ] = array_628hc5a1

    candidate_rows_628hc5a1.append(
        {
            "name": name_628hc5a1,
            "exists": True,
            "shape": str(array_628hc5a1.shape),
            "finite": finite_628hc5a1,
            "min": float(np.min(array_628hc5a1)),
            "median": float(np.median(array_628hc5a1)),
            "mean": float(np.mean(array_628hc5a1)),
            "max": float(np.max(array_628hc5a1)),
            "within_0_1": within_0_1_628hc5a1,
        }
    )

FD002_C5A1_TEST_CANDIDATE_SUMMARY_628HC5A1 = (
    pd.DataFrame(
        candidate_rows_628hc5a1
    )
)

print(
    "\n--- EXISTING TEST UNCERTAINTY/RISK CANDIDATES ---"
)

display(
    FD002_C5A1_TEST_CANDIDATE_SUMMARY_628HC5A1
)

# ---------------------------------------------------------------
# 4. Pairwise exact / numerical equality
# ---------------------------------------------------------------

existing_names_628hc5a1 = list(
    candidate_arrays_628hc5a1.keys()
)

pair_rows_628hc5a1 = []

for index_a_628hc5a1 in range(
    len(existing_names_628hc5a1)
):

    name_a_628hc5a1 = (
        existing_names_628hc5a1[
            index_a_628hc5a1
        ]
    )

    array_a_628hc5a1 = (
        candidate_arrays_628hc5a1[
            name_a_628hc5a1
        ]
    )

    for index_b_628hc5a1 in range(
        index_a_628hc5a1 + 1,
        len(existing_names_628hc5a1),
    ):

        name_b_628hc5a1 = (
            existing_names_628hc5a1[
                index_b_628hc5a1
            ]
        )

        array_b_628hc5a1 = (
            candidate_arrays_628hc5a1[
                name_b_628hc5a1
            ]
        )

        max_abs_error_628hc5a1 = float(
            np.max(
                np.abs(
                    array_a_628hc5a1
                    -
                    array_b_628hc5a1
                )
            )
        )

        pair_rows_628hc5a1.append(
            {
                "object_a":
                    name_a_628hc5a1,

                "object_b":
                    name_b_628hc5a1,

                "array_equal":
                    bool(
                        np.array_equal(
                            array_a_628hc5a1,
                            array_b_628hc5a1,
                        )
                    ),

                "allclose_1e12":
                    bool(
                        np.allclose(
                            array_a_628hc5a1,
                            array_b_628hc5a1,
                            rtol=0.0,
                            atol=1e-12,
                        )
                    ),

                "max_abs_error":
                    max_abs_error_628hc5a1,

                "spearman":
                    float(
                        pd.Series(
                            array_a_628hc5a1
                        )
                        .corr(
                            pd.Series(
                                array_b_628hc5a1
                            ),
                            method="spearman",
                        )
                    ),
            }
        )

FD002_C5A1_PAIRWISE_RECONCILIATION_628HC5A1 = (
    pd.DataFrame(
        pair_rows_628hc5a1
    )
)

print(
    "\n--- EXACTLY IDENTICAL TEST VECTOR PAIRS ---"
)

exact_pairs_628hc5a1 = (
    FD002_C5A1_PAIRWISE_RECONCILIATION_628HC5A1[
        FD002_C5A1_PAIRWISE_RECONCILIATION_628HC5A1[
            "array_equal"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    exact_pairs_628hc5a1
)

# ---------------------------------------------------------------
# 5. Focused historical lineage checks
# ---------------------------------------------------------------

required_focus_names_628hc5a1 = [
    "test_u",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB",
    "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",
    "FD002_TEST_SELECTIVE_RISK_628FC",
    "FD002_TEST_SELECTIVE_RISK_628FCR2",
]

missing_focus_628hc5a1 = [
    name_628hc5a1
    for name_628hc5a1
    in required_focus_names_628hc5a1
    if name_628hc5a1
    not in candidate_arrays_628hc5a1
]

if missing_focus_628hc5a1:
    raise RuntimeError(
        "Required historical TEST lineage objects are missing: "
        +
        ", ".join(
            missing_focus_628hc5a1
        )
    )

focus_rows_628hc5a1 = []

focus_pairs_628hc5a1 = [
    (
        "FD002_TEST_ADJUSTED_UNCERTAINTY_628FB",
        "FD002_TEST_ADJUSTED_UNCERTAINTY_628FBR1",
    ),
    (
        "FD002_TEST_G1_UNCERTAINTY_RISK_628FC",
        "FD002_TEST_SELECTIVE_RISK_628FC",
    ),
    (
        "FD002_TEST_SELECTIVE_RISK_628FC",
        "FD002_TEST_SELECTIVE_RISK_628FCR2",
    ),
]

for (
    name_a_628hc5a1,
    name_b_628hc5a1,
) in focus_pairs_628hc5a1:

    a_628hc5a1 = candidate_arrays_628hc5a1[
        name_a_628hc5a1
    ]

    b_628hc5a1 = candidate_arrays_628hc5a1[
        name_b_628hc5a1
    ]

    focus_rows_628hc5a1.append(
        {
            "object_a": name_a_628hc5a1,
            "object_b": name_b_628hc5a1,

            "array_equal":
                bool(
                    np.array_equal(
                        a_628hc5a1,
                        b_628hc5a1,
                    )
                ),

            "max_abs_error":
                float(
                    np.max(
                        np.abs(
                            a_628hc5a1
                            -
                            b_628hc5a1
                        )
                    )
                ),

            "spearman":
                float(
                    pd.Series(
                        a_628hc5a1
                    )
                    .corr(
                        pd.Series(
                            b_628hc5a1
                        ),
                        method="spearman",
                    )
                ),
        }
    )

FD002_C5A1_FOCUSED_LINEAGE_628HC5A1 = (
    pd.DataFrame(
        focus_rows_628hc5a1
    )
)

print(
    "\n--- FOCUSED HISTORICAL TEST LINEAGE ---"
)

display(
    FD002_C5A1_FOCUSED_LINEAGE_628HC5A1
)

# ---------------------------------------------------------------
# 6. Identify risk-like candidates only
#
# This is diagnostic only.
# Nothing is selected yet.
# ---------------------------------------------------------------

risk_like_rows_628hc5a1 = []

for name_628hc5a1 in existing_names_628hc5a1:

    array_628hc5a1 = (
        candidate_arrays_628hc5a1[
            name_628hc5a1
        ]
    )

    if (
        np.min(array_628hc5a1) >= 0.0
        and
        np.max(array_628hc5a1) <= 1.0
    ):

        risk_like_rows_628hc5a1.append(
            {
                "name":
                    name_628hc5a1,

                "min":
                    float(
                        np.min(
                            array_628hc5a1
                        )
                    ),

                "median":
                    float(
                        np.median(
                            array_628hc5a1
                        )
                    ),

                "mean":
                    float(
                        np.mean(
                            array_628hc5a1
                        )
                    ),

                "max":
                    float(
                        np.max(
                            array_628hc5a1
                        )
                    ),

                "fraction_le_frozen_threshold":
                    float(
                        np.mean(
                            array_628hc5a1
                            <=
                            frozen_threshold_628hc5a1
                        )
                    ),
            }
        )

FD002_C5A1_RISK_LIKE_CANDIDATES_628HC5A1 = (
    pd.DataFrame(
        risk_like_rows_628hc5a1
    )
    .sort_values(
        "name",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "\n--- RISK-LIKE 259-ENGINE CANDIDATES ---"
)

display(
    FD002_C5A1_RISK_LIKE_CANDIDATES_628HC5A1
)

# ---------------------------------------------------------------
# 7. Audit
# ---------------------------------------------------------------

audit_628hc5a1 = pd.DataFrame(
    {
        "check": [
            "frozen_architecture_unchanged",
            "frozen_policy_unchanged",
            "frozen_coverage_080",
            "frozen_threshold_exact",
            "official_test_label_marker_false",
            "required_historical_test_objects_present",
            "all_existing_candidates_length_259",
            "all_existing_candidates_finite",
            "no_test_file_read",
            "no_model_execution",
            "no_test_scoring",
            "no_threshold_application_decision",
            "no_files_written",
        ],

        "passed": [
            frozen_architecture_628hc5a1
            ==
            "uncertainty_only",

            frozen_policy_628hc5a1
            ==
            "uncertainty_only_fixed_threshold",

            frozen_coverage_628hc5a1
            ==
            0.80,

            frozen_threshold_628hc5a1
            ==
            0.8003345892330226,

            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            )
            is False,

            len(
                missing_focus_628hc5a1
            )
            ==
            0,

            all(
                array_628hc5a1.shape
                ==
                (259,)
                for array_628hc5a1
                in candidate_arrays_628hc5a1.values()
            ),

            all(
                np.isfinite(
                    array_628hc5a1
                ).all()
                for array_628hc5a1
                in candidate_arrays_628hc5a1.values()
            ),

            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print(
    "\n--- C5-A1 AUDIT ---"
)

display(
    audit_628hc5a1
)

all_checks_passed_628hc5a1 = bool(
    audit_628hc5a1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a1:
    raise RuntimeError(
        "C5-A1 TEST uncertainty-risk lineage reconciliation failed."
    )

# ---------------------------------------------------------------
# 8. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C5-A1 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Frozen reliability architecture :",
    frozen_architecture_628hc5a1,
)

print(
    "Frozen operating policy         :",
    frozen_policy_628hc5a1,
)

print(
    "Frozen exact threshold          :",
    repr(
        frozen_threshold_628hc5a1
    ),
)

print(
    "Historical candidates found     :",
    len(
        candidate_arrays_628hc5a1
    ),
)

print(
    "Risk-like candidates found      :",
    len(
        FD002_C5A1_RISK_LIKE_CANDIDATES_628HC5A1
    ),
)

print(
    "Exact-equality pairs found      :",
    len(
        exact_pairs_628hc5a1
    ),
)

print(
    "All checks passed               :",
    all_checks_passed_628hc5a1,
)

print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Frozen threshold modified        : False")
print("Frozen threshold applied         : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files written                    : False")

print(
    "\n=== END STEP 6.28H-C5-A1 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-A2
# CANONICAL FD002 TEST RELIABILITY-RISK IDENTITY LOCK
# ================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A2 — CANONICAL FD002 TEST RELIABILITY-RISK IDENTITY LOCK")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconfirm frozen C4 development decision
# ---------------------------------------------------------------

frozen_architecture_628hc5a2 = str(
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B[
        "reliability_architecture"
    ]
)

frozen_policy_628hc5a2 = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

frozen_coverage_628hc5a2 = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

frozen_threshold_628hc5a2 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

if frozen_architecture_628hc5a2 != "uncertainty_only":
    raise RuntimeError(
        "Frozen reliability architecture is no longer uncertainty_only."
    )

if frozen_policy_628hc5a2 != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Frozen operating policy changed unexpectedly."
    )

if frozen_coverage_628hc5a2 != 0.80:
    raise RuntimeError(
        "Frozen nominal coverage changed unexpectedly."
    )

if frozen_threshold_628hc5a2 != 0.8003345892330226:
    raise RuntimeError(
        "Frozen C4 threshold changed unexpectedly."
    )

# ---------------------------------------------------------------
# 2. TEST-label safety barrier
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Historical TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is True. Stop."
    )

# ---------------------------------------------------------------
# 3. Explicit corrected final-risk lineage
# ---------------------------------------------------------------

canonical_source_628hc5a2 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

stored_final_628hc5a2 = np.asarray(
    stored_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

reproduced_final_628hc5a2 = np.asarray(
    reproduced_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

historical_test_final_628hc5a2 = np.asarray(
    test_final,
    dtype=np.float64,
).reshape(-1)

# Historical pre-repair object: audit only.
old_fc_risk_628hc5a2 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FC,
    dtype=np.float64,
).reshape(-1)

# ---------------------------------------------------------------
# 4. Shape / validity contract
# ---------------------------------------------------------------

expected_shape_628hc5a2 = (259,)

arrays_628hc5a2 = {
    "FD002_TEST_SELECTIVE_RISK_628FCR2":
        canonical_source_628hc5a2,

    "stored_test_final_risk_628hc0r2":
        stored_final_628hc5a2,

    "reproduced_test_final_risk_628hc0r2":
        reproduced_final_628hc5a2,

    "test_final":
        historical_test_final_628hc5a2,
}

contract_rows_628hc5a2 = []

for name_628hc5a2, arr_628hc5a2 in arrays_628hc5a2.items():

    shape_ok_628hc5a2 = bool(
        arr_628hc5a2.shape
        ==
        expected_shape_628hc5a2
    )

    finite_ok_628hc5a2 = bool(
        np.isfinite(
            arr_628hc5a2
        ).all()
    )

    range_ok_628hc5a2 = bool(
        finite_ok_628hc5a2
        and
        np.min(arr_628hc5a2) >= 0.0
        and
        np.max(arr_628hc5a2) <= 1.0
    )

    contract_rows_628hc5a2.append(
        {
            "name": name_628hc5a2,
            "shape": str(arr_628hc5a2.shape),
            "shape_ok": shape_ok_628hc5a2,
            "finite": finite_ok_628hc5a2,
            "within_0_1": range_ok_628hc5a2,
            "min": float(np.min(arr_628hc5a2)),
            "median": float(np.median(arr_628hc5a2)),
            "mean": float(np.mean(arr_628hc5a2)),
            "max": float(np.max(arr_628hc5a2)),
        }
    )

FD002_C5A2_FINAL_RISK_CONTRACT_628HC5A2 = pd.DataFrame(
    contract_rows_628hc5a2
)

print("\n--- CORRECTED FINAL-RISK LINEAGE CONTRACT ---")
display(
    FD002_C5A2_FINAL_RISK_CONTRACT_628HC5A2
)

# ---------------------------------------------------------------
# 5. Exact identity verification
# ---------------------------------------------------------------

canonical_vs_stored_628hc5a2 = bool(
    np.array_equal(
        canonical_source_628hc5a2,
        stored_final_628hc5a2,
    )
)

canonical_vs_reproduced_628hc5a2 = bool(
    np.array_equal(
        canonical_source_628hc5a2,
        reproduced_final_628hc5a2,
    )
)

canonical_vs_test_final_628hc5a2 = bool(
    np.array_equal(
        canonical_source_628hc5a2,
        historical_test_final_628hc5a2,
    )
)

stored_vs_reproduced_628hc5a2 = bool(
    np.array_equal(
        stored_final_628hc5a2,
        reproduced_final_628hc5a2,
    )
)

# Old FC object must remain distinguishable.
canonical_differs_from_old_fc_628hc5a2 = bool(
    not np.array_equal(
        canonical_source_628hc5a2,
        old_fc_risk_628hc5a2,
    )
)

old_fc_max_abs_difference_628hc5a2 = float(
    np.max(
        np.abs(
            canonical_source_628hc5a2
            -
            old_fc_risk_628hc5a2
        )
    )
)

identity_table_628hc5a2 = pd.DataFrame(
    {
        "check": [
            "FCR2_exact_vs_stored_final",
            "FCR2_exact_vs_reproduced_final",
            "FCR2_exact_vs_test_final",
            "stored_exact_vs_reproduced",
            "FCR2_distinct_from_old_FC",
        ],

        "passed": [
            canonical_vs_stored_628hc5a2,
            canonical_vs_reproduced_628hc5a2,
            canonical_vs_test_final_628hc5a2,
            stored_vs_reproduced_628hc5a2,
            canonical_differs_from_old_fc_628hc5a2,
        ],
    }
)

print("\n--- FINAL-RISK IDENTITY AUDIT ---")
display(
    identity_table_628hc5a2
)

print(
    "\nOld FC vs corrected FCR2 max abs difference:",
    old_fc_max_abs_difference_628hc5a2,
)

# ---------------------------------------------------------------
# 6. Canonical hash
# ---------------------------------------------------------------

def sha256_array_628hc5a2(arr):
    contiguous = np.ascontiguousarray(
        arr,
        dtype=np.float64,
    )

    return hashlib.sha256(
        contiguous.tobytes(
            order="C"
        )
    ).hexdigest()


canonical_hash_628hc5a2 = (
    sha256_array_628hc5a2(
        canonical_source_628hc5a2
    )
)

stored_hash_628hc5a2 = (
    sha256_array_628hc5a2(
        stored_final_628hc5a2
    )
)

reproduced_hash_628hc5a2 = (
    sha256_array_628hc5a2(
        reproduced_final_628hc5a2
    )
)

print("\n--- SHA-256 ---")
print(
    "FCR2 canonical :",
    canonical_hash_628hc5a2,
)
print(
    "Stored final   :",
    stored_hash_628hc5a2,
)
print(
    "Reproduced     :",
    reproduced_hash_628hc5a2,
)

# ---------------------------------------------------------------
# 7. Create canonical in-memory TEST risk object
#
# Important:
# This is an identity alias/copy only.
# No threshold is applied here.
# ---------------------------------------------------------------

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2 = (
    canonical_source_628hc5a2.copy()
)

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2 = (
    "FD002_TEST_SELECTIVE_RISK_628FCR2"
)

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2 = (
    canonical_hash_628hc5a2
)

# ---------------------------------------------------------------
# 8. Audit
# ---------------------------------------------------------------

all_contracts_valid_628hc5a2 = bool(
    FD002_C5A2_FINAL_RISK_CONTRACT_628HC5A2[
        [
            "shape_ok",
            "finite",
            "within_0_1",
        ]
    ]
    .to_numpy(
        dtype=bool
    )
    .all()
)

all_identity_checks_628hc5a2 = bool(
    identity_table_628hc5a2[
        "passed"
    ].all()
)

hashes_exact_628hc5a2 = bool(
    canonical_hash_628hc5a2
    ==
    stored_hash_628hc5a2
    ==
    reproduced_hash_628hc5a2
)

audit_628hc5a2 = pd.DataFrame(
    {
        "check": [
            "frozen_architecture_uncertainty_only",
            "frozen_policy_exact",
            "frozen_nominal_coverage_080",
            "frozen_threshold_exact",
            "official_test_label_marker_false",
            "corrected_final_risk_contract_valid",
            "corrected_final_lineage_exact",
            "corrected_final_hashes_exact",
            "old_FC_distinct_from_corrected_FCR2",
            "canonical_test_risk_shape_259",
            "no_test_file_read",
            "no_test_label_read",
            "no_model_execution",
            "no_test_scoring",
            "no_threshold_application",
            "no_files_written",
        ],

        "passed": [
            frozen_architecture_628hc5a2
            ==
            "uncertainty_only",

            frozen_policy_628hc5a2
            ==
            "uncertainty_only_fixed_threshold",

            frozen_coverage_628hc5a2
            ==
            0.80,

            frozen_threshold_628hc5a2
            ==
            0.8003345892330226,

            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            )
            is False,

            all_contracts_valid_628hc5a2,

            all_identity_checks_628hc5a2,

            hashes_exact_628hc5a2,

            canonical_differs_from_old_fc_628hc5a2,

            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2.shape
            ==
            (259,),

            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A2 AUDIT ---")
display(
    audit_628hc5a2
)

all_checks_passed_628hc5a2 = bool(
    audit_628hc5a2[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a2:
    raise RuntimeError(
        "C5-A2 canonical TEST-risk identity lock failed."
    )

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A2 FINAL SUMMARY")
print("=" * 94)

print(
    "Frozen reliability architecture :",
    frozen_architecture_628hc5a2,
)

print(
    "Canonical TEST risk source       :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2,
)

print(
    "Canonical TEST risk shape        :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2.shape,
)

print(
    "Canonical TEST risk min/median/mean/max :",
    float(
        np.min(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2
        )
    ),
    float(
        np.median(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2
        )
    ),
    float(
        np.mean(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2
        )
    ),
    float(
        np.max(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2
        )
    ),
)

print(
    "Canonical SHA-256               :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2,
)

print(
    "Corrected final lineage exact    :",
    all_identity_checks_628hc5a2,
)

print(
    "Old FC risk rejected as canonical:",
    canonical_differs_from_old_fc_628hc5a2,
)

print(
    "All checks passed               :",
    all_checks_passed_628hc5a2,
)

print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Frozen threshold modified        : False")
print("Frozen threshold applied         : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files written                    : False")

print("\n=== END STEP 6.28H-C5-A2 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A2-R1
# REPRODUCED-FINAL NUMERICAL DISCREPANCY AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A2-R1 — REPRODUCED-FINAL NUMERICAL DISCREPANCY AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Explicit historical final-risk vectors
# ---------------------------------------------------------------

fcr2_628hc5a2r1 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

stored_628hc5a2r1 = np.asarray(
    stored_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

reproduced_628hc5a2r1 = np.asarray(
    reproduced_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

test_final_628hc5a2r1 = np.asarray(
    test_final,
    dtype=np.float64,
).reshape(-1)

frozen_threshold_628hc5a2r1 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

expected_shape_628hc5a2r1 = (259,)

# ---------------------------------------------------------------
# 2. Basic safety contract
# ---------------------------------------------------------------

shape_contract_628hc5a2r1 = bool(
    fcr2_628hc5a2r1.shape
    ==
    stored_628hc5a2r1.shape
    ==
    reproduced_628hc5a2r1.shape
    ==
    test_final_628hc5a2r1.shape
    ==
    expected_shape_628hc5a2r1
)

finite_contract_628hc5a2r1 = bool(
    np.isfinite(fcr2_628hc5a2r1).all()
    and
    np.isfinite(stored_628hc5a2r1).all()
    and
    np.isfinite(reproduced_628hc5a2r1).all()
    and
    np.isfinite(test_final_628hc5a2r1).all()
)

if not shape_contract_628hc5a2r1:
    raise RuntimeError(
        "One or more TEST final-risk vectors do not have shape (259,)."
    )

if not finite_contract_628hc5a2r1:
    raise RuntimeError(
        "One or more TEST final-risk vectors contain non-finite values."
    )

# ---------------------------------------------------------------
# 3. Authoritative historical-cluster check
#
# This does NOT declare a canonical vector yet.
# It only verifies the exact historical agreement already observed.
# ---------------------------------------------------------------

fcr2_vs_stored_628hc5a2r1 = bool(
    np.array_equal(
        fcr2_628hc5a2r1,
        stored_628hc5a2r1,
    )
)

fcr2_vs_test_final_628hc5a2r1 = bool(
    np.array_equal(
        fcr2_628hc5a2r1,
        test_final_628hc5a2r1,
    )
)

authoritative_cluster_exact_628hc5a2r1 = bool(
    fcr2_vs_stored_628hc5a2r1
    and
    fcr2_vs_test_final_628hc5a2r1
)

print("\n--- EXACT HISTORICAL CLUSTER ---")
print(
    "FCR2 exact vs stored final :",
    fcr2_vs_stored_628hc5a2r1,
)
print(
    "FCR2 exact vs test_final   :",
    fcr2_vs_test_final_628hc5a2r1,
)
print(
    "Three-object cluster exact :",
    authoritative_cluster_exact_628hc5a2r1,
)

# ---------------------------------------------------------------
# 4. Reproduced-vs-FCR2 numerical discrepancy
# ---------------------------------------------------------------

signed_diff_628hc5a2r1 = (
    reproduced_628hc5a2r1
    -
    fcr2_628hc5a2r1
)

abs_diff_628hc5a2r1 = np.abs(
    signed_diff_628hc5a2r1
)

different_mask_628hc5a2r1 = (
    reproduced_628hc5a2r1
    !=
    fcr2_628hc5a2r1
)

different_count_628hc5a2r1 = int(
    np.count_nonzero(
        different_mask_628hc5a2r1
    )
)

max_abs_diff_628hc5a2r1 = float(
    np.max(
        abs_diff_628hc5a2r1
    )
)

mean_abs_diff_628hc5a2r1 = float(
    np.mean(
        abs_diff_628hc5a2r1
    )
)

median_abs_diff_628hc5a2r1 = float(
    np.median(
        abs_diff_628hc5a2r1
    )
)

q95_abs_diff_628hc5a2r1 = float(
    np.quantile(
        abs_diff_628hc5a2r1,
        0.95,
    )
)

rmse_diff_628hc5a2r1 = float(
    np.sqrt(
        np.mean(
            signed_diff_628hc5a2r1 ** 2
        )
    )
)

print("\n--- REPRODUCED VS FCR2 NUMERICAL DIFFERENCE ---")

print(
    "Different elements :",
    different_count_628hc5a2r1,
    "/",
    len(fcr2_628hc5a2r1),
)

print(
    "Max abs difference :",
    repr(max_abs_diff_628hc5a2r1),
)

print(
    "Mean abs difference:",
    repr(mean_abs_diff_628hc5a2r1),
)

print(
    "Median abs diff    :",
    repr(median_abs_diff_628hc5a2r1),
)

print(
    "q95 abs difference :",
    repr(q95_abs_diff_628hc5a2r1),
)

print(
    "RMSE difference    :",
    repr(rmse_diff_628hc5a2r1),
)

# ---------------------------------------------------------------
# 5. Numerical-closeness ladder
# ---------------------------------------------------------------

tolerances_628hc5a2r1 = [
    0.0,
    1e-15,
    1e-14,
    1e-13,
    1e-12,
    1e-11,
    1e-10,
    1e-9,
    1e-8,
    1e-7,
    1e-6,
]

closeness_rows_628hc5a2r1 = []

for atol_628hc5a2r1 in tolerances_628hc5a2r1:

    closeness_rows_628hc5a2r1.append(
        {
            "atol":
                atol_628hc5a2r1,

            "allclose_rtol0":
                bool(
                    np.allclose(
                        fcr2_628hc5a2r1,
                        reproduced_628hc5a2r1,
                        rtol=0.0,
                        atol=atol_628hc5a2r1,
                    )
                ),

            "fraction_absdiff_le_atol":
                float(
                    np.mean(
                        abs_diff_628hc5a2r1
                        <=
                        atol_628hc5a2r1
                    )
                ),
        }
    )

FD002_C5A2R1_CLOSENESS_628HC5A2R1 = pd.DataFrame(
    closeness_rows_628hc5a2r1
)

print("\n--- NUMERICAL CLOSENESS GRID ---")
display(
    FD002_C5A2R1_CLOSENESS_628HC5A2R1
)

# ---------------------------------------------------------------
# 6. Ranking agreement
# ---------------------------------------------------------------

spearman_628hc5a2r1 = float(
    pd.Series(
        fcr2_628hc5a2r1
    ).corr(
        pd.Series(
            reproduced_628hc5a2r1
        ),
        method="spearman",
    )
)

pearson_628hc5a2r1 = float(
    np.corrcoef(
        fcr2_628hc5a2r1,
        reproduced_628hc5a2r1,
    )[0, 1]
)

fcr2_order_628hc5a2r1 = np.argsort(
    fcr2_628hc5a2r1,
    kind="mergesort",
)

reproduced_order_628hc5a2r1 = np.argsort(
    reproduced_628hc5a2r1,
    kind="mergesort",
)

exact_sorted_order_628hc5a2r1 = bool(
    np.array_equal(
        fcr2_order_628hc5a2r1,
        reproduced_order_628hc5a2r1,
    )
)

print("\n--- ORDERING AGREEMENT ---")

print(
    "Spearman correlation :",
    repr(spearman_628hc5a2r1),
)

print(
    "Pearson correlation  :",
    repr(pearson_628hc5a2r1),
)

print(
    "Exact sorted order   :",
    exact_sorted_order_628hc5a2r1,
)

# ---------------------------------------------------------------
# 7. Frozen-threshold agreement
#
# IMPORTANT:
# This is ONLY a consistency diagnostic.
#
# The threshold was already frozen entirely on DEV.
# This section does NOT tune, modify, or select a threshold.
# No TEST labels are used.
# ---------------------------------------------------------------

fcr2_accept_628hc5a2r1 = (
    fcr2_628hc5a2r1
    <=
    frozen_threshold_628hc5a2r1
)

reproduced_accept_628hc5a2r1 = (
    reproduced_628hc5a2r1
    <=
    frozen_threshold_628hc5a2r1
)

decision_disagreements_628hc5a2r1 = int(
    np.count_nonzero(
        fcr2_accept_628hc5a2r1
        !=
        reproduced_accept_628hc5a2r1
    )
)

fcr2_accepted_count_628hc5a2r1 = int(
    np.count_nonzero(
        fcr2_accept_628hc5a2r1
    )
)

reproduced_accepted_count_628hc5a2r1 = int(
    np.count_nonzero(
        reproduced_accept_628hc5a2r1
    )
)

fcr2_coverage_628hc5a2r1 = float(
    np.mean(
        fcr2_accept_628hc5a2r1
    )
)

reproduced_coverage_628hc5a2r1 = float(
    np.mean(
        reproduced_accept_628hc5a2r1
    )
)

print("\n--- FROZEN-THRESHOLD DECISION AGREEMENT ---")

print(
    "Frozen threshold             :",
    repr(
        frozen_threshold_628hc5a2r1
    ),
)

print(
    "FCR2 accepted engines        :",
    fcr2_accepted_count_628hc5a2r1,
)

print(
    "Reproduced accepted engines  :",
    reproduced_accepted_count_628hc5a2r1,
)

print(
    "FCR2 realized coverage       :",
    repr(
        fcr2_coverage_628hc5a2r1
    ),
)

print(
    "Reproduced realized coverage :",
    repr(
        reproduced_coverage_628hc5a2r1
    ),
)

print(
    "Acceptance disagreements     :",
    decision_disagreements_628hc5a2r1,
)

# ---------------------------------------------------------------
# 8. Largest element-level discrepancies
# ---------------------------------------------------------------

largest_indices_628hc5a2r1 = np.argsort(
    -abs_diff_628hc5a2r1,
    kind="mergesort",
)[:15]

largest_rows_628hc5a2r1 = []

for index_628hc5a2r1 in largest_indices_628hc5a2r1:

    largest_rows_628hc5a2r1.append(
        {
            "engine_position_1based":
                int(
                    index_628hc5a2r1
                    +
                    1
                ),

            "FCR2":
                float(
                    fcr2_628hc5a2r1[
                        index_628hc5a2r1
                    ]
                ),

            "reproduced":
                float(
                    reproduced_628hc5a2r1[
                        index_628hc5a2r1
                    ]
                ),

            "abs_difference":
                float(
                    abs_diff_628hc5a2r1[
                        index_628hc5a2r1
                    ]
                ),

            "FCR2_accept":
                bool(
                    fcr2_accept_628hc5a2r1[
                        index_628hc5a2r1
                    ]
                ),

            "reproduced_accept":
                bool(
                    reproduced_accept_628hc5a2r1[
                        index_628hc5a2r1
                    ]
                ),
        }
    )

FD002_C5A2R1_TOP_DIFFERENCES_628HC5A2R1 = pd.DataFrame(
    largest_rows_628hc5a2r1
)

print("\n--- TOP 15 ABSOLUTE DISCREPANCIES ---")
display(
    FD002_C5A2R1_TOP_DIFFERENCES_628HC5A2R1
)

# ---------------------------------------------------------------
# 9. Audit
#
# Reproduction exactness is intentionally NOT required here.
# That is the quantity being diagnosed.
# ---------------------------------------------------------------

audit_628hc5a2r1 = pd.DataFrame(
    {
        "check": [
            "all_shapes_259",
            "all_values_finite",
            "FCR2_exact_vs_stored_final",
            "FCR2_exact_vs_test_final",
            "historical_three_object_cluster_exact",
            "official_test_label_marker_false",
            "no_test_file_read",
            "no_test_label_read",
            "no_model_execution",
            "no_model_retraining",
            "no_test_scoring",
            "no_threshold_tuning",
            "no_files_written",
        ],

        "passed": [
            shape_contract_628hc5a2r1,
            finite_contract_628hc5a2r1,
            fcr2_vs_stored_628hc5a2r1,
            fcr2_vs_test_final_628hc5a2r1,
            authoritative_cluster_exact_628hc5a2r1,

            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            )
            is False,

            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A2-R1 AUDIT ---")
display(
    audit_628hc5a2r1
)

all_checks_passed_628hc5a2r1 = bool(
    audit_628hc5a2r1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a2r1:

    raise RuntimeError(
        "C5-A2-R1 historical-cluster audit failed."
    )

# ---------------------------------------------------------------
# 10. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A2-R1 FINAL SUMMARY")
print("=" * 94)

print(
    "Exact historical cluster        : "
    "FCR2 == stored_final == test_final"
)

print(
    "Historical cluster exact        :",
    authoritative_cluster_exact_628hc5a2r1,
)

print(
    "Reproduced exact vs FCR2        :",
    bool(
        np.array_equal(
            reproduced_628hc5a2r1,
            fcr2_628hc5a2r1,
        )
    ),
)

print(
    "Reproduced max abs difference   :",
    repr(
        max_abs_diff_628hc5a2r1
    ),
)

print(
    "Reproduced mean abs difference  :",
    repr(
        mean_abs_diff_628hc5a2r1
    ),
)

print(
    "Reproduced Spearman vs FCR2     :",
    repr(
        spearman_628hc5a2r1
    ),
)

print(
    "Frozen-threshold disagreements  :",
    decision_disagreements_628hc5a2r1,
)

print(
    "All required checks passed      :",
    all_checks_passed_628hc5a2r1,
)

print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Threshold tuned or modified      : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files written                    : False")

print("\n=== END STEP 6.28H-C5-A2-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A2-R2
# FINAL CANONICAL FD002 TEST RELIABILITY-RISK IDENTITY LOCK
# ================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A2-R2 — FINAL CANONICAL FD002 TEST RELIABILITY-RISK IDENTITY LOCK")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconfirm frozen development contract
# ---------------------------------------------------------------

frozen_architecture_628hc5a2r2 = str(
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B[
        "reliability_architecture"
    ]
)

frozen_policy_628hc5a2r2 = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

frozen_coverage_628hc5a2r2 = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

frozen_threshold_628hc5a2r2 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

# ---------------------------------------------------------------
# 2. Explicit corrected historical cluster + reproduction
# ---------------------------------------------------------------

fcr2_628hc5a2r2 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

stored_628hc5a2r2 = np.asarray(
    stored_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

test_final_628hc5a2r2 = np.asarray(
    test_final,
    dtype=np.float64,
).reshape(-1)

reproduced_628hc5a2r2 = np.asarray(
    reproduced_test_final_risk_628hc0r2,
    dtype=np.float64,
).reshape(-1)

expected_shape_628hc5a2r2 = (259,)

# ---------------------------------------------------------------
# 3. Exact historical identity contract
# ---------------------------------------------------------------

shape_contract_628hc5a2r2 = bool(
    fcr2_628hc5a2r2.shape
    ==
    stored_628hc5a2r2.shape
    ==
    test_final_628hc5a2r2.shape
    ==
    reproduced_628hc5a2r2.shape
    ==
    expected_shape_628hc5a2r2
)

finite_contract_628hc5a2r2 = bool(
    np.isfinite(fcr2_628hc5a2r2).all()
    and
    np.isfinite(stored_628hc5a2r2).all()
    and
    np.isfinite(test_final_628hc5a2r2).all()
    and
    np.isfinite(reproduced_628hc5a2r2).all()
)

range_contract_628hc5a2r2 = bool(
    np.min(fcr2_628hc5a2r2) >= 0.0
    and
    np.max(fcr2_628hc5a2r2) <= 1.0
    and
    np.min(stored_628hc5a2r2) >= 0.0
    and
    np.max(stored_628hc5a2r2) <= 1.0
    and
    np.min(test_final_628hc5a2r2) >= 0.0
    and
    np.max(test_final_628hc5a2r2) <= 1.0
    and
    np.min(reproduced_628hc5a2r2) >= 0.0
    and
    np.max(reproduced_628hc5a2r2) <= 1.0
)

historical_cluster_exact_628hc5a2r2 = bool(
    np.array_equal(
        fcr2_628hc5a2r2,
        stored_628hc5a2r2,
    )
    and
    np.array_equal(
        fcr2_628hc5a2r2,
        test_final_628hc5a2r2,
    )
)

# ---------------------------------------------------------------
# 4. Numerical-reproduction contract
# ---------------------------------------------------------------

reproduction_atol_628hc5a2r2 = 1e-14

reproduction_close_628hc5a2r2 = bool(
    np.allclose(
        fcr2_628hc5a2r2,
        reproduced_628hc5a2r2,
        rtol=0.0,
        atol=reproduction_atol_628hc5a2r2,
    )
)

reproduction_max_abs_diff_628hc5a2r2 = float(
    np.max(
        np.abs(
            fcr2_628hc5a2r2
            -
            reproduced_628hc5a2r2
        )
    )
)

reproduction_order_exact_628hc5a2r2 = bool(
    np.array_equal(
        np.argsort(
            fcr2_628hc5a2r2,
            kind="mergesort",
        ),
        np.argsort(
            reproduced_628hc5a2r2,
            kind="mergesort",
        ),
    )
)

fcr2_accept_diag_628hc5a2r2 = (
    fcr2_628hc5a2r2
    <=
    frozen_threshold_628hc5a2r2
)

reproduced_accept_diag_628hc5a2r2 = (
    reproduced_628hc5a2r2
    <=
    frozen_threshold_628hc5a2r2
)

reproduction_threshold_disagreements_628hc5a2r2 = int(
    np.count_nonzero(
        fcr2_accept_diag_628hc5a2r2
        !=
        reproduced_accept_diag_628hc5a2r2
    )
)

reproduction_decision_equivalent_628hc5a2r2 = bool(
    reproduction_threshold_disagreements_628hc5a2r2
    ==
    0
)

# ---------------------------------------------------------------
# 5. Canonical source hash
# ---------------------------------------------------------------

def sha256_array_628hc5a2r2(arr):

    contiguous_628hc5a2r2 = np.ascontiguousarray(
        arr,
        dtype=np.float64,
    )

    return hashlib.sha256(
        contiguous_628hc5a2r2.tobytes(
            order="C"
        )
    ).hexdigest()


canonical_hash_628hc5a2r2 = (
    sha256_array_628hc5a2r2(
        fcr2_628hc5a2r2
    )
)

stored_hash_628hc5a2r2 = (
    sha256_array_628hc5a2r2(
        stored_628hc5a2r2
    )
)

test_final_hash_628hc5a2r2 = (
    sha256_array_628hc5a2r2(
        test_final_628hc5a2r2
    )
)

historical_hash_cluster_exact_628hc5a2r2 = bool(
    canonical_hash_628hc5a2r2
    ==
    stored_hash_628hc5a2r2
    ==
    test_final_hash_628hc5a2r2
)

# ---------------------------------------------------------------
# 6. Final canonical in-memory TEST reliability-risk object
#
# Identity lock only.
# No operational TEST mask is frozen here.
# ---------------------------------------------------------------

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2 = (
    fcr2_628hc5a2r2.copy()
)

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2R2 = (
    "FD002_TEST_SELECTIVE_RISK_628FCR2"
)

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2R2 = (
    canonical_hash_628hc5a2r2
)

FD002_CANONICAL_TEST_UNCERTAINTY_RISK_LINEAGE_628HC5A2R2 = {
    "dataset": "FD002",
    "population": "all_259_engines",

    "reliability_architecture":
        "uncertainty_only",

    "canonical_source":
        "FD002_TEST_SELECTIVE_RISK_628FCR2",

    "exact_historical_equivalents": [
        "stored_test_final_risk_628hc0r2",
        "test_final",
    ],

    "numerically_reproduced_source":
        "reproduced_test_final_risk_628hc0r2",

    "reproduction_atol":
        reproduction_atol_628hc5a2r2,

    "reproduction_max_abs_difference":
        reproduction_max_abs_diff_628hc5a2r2,

    "reproduction_order_exact":
        reproduction_order_exact_628hc5a2r2,

    "reproduction_frozen_threshold_disagreements":
        reproduction_threshold_disagreements_628hc5a2r2,

    "canonical_sha256":
        canonical_hash_628hc5a2r2,
}

# ---------------------------------------------------------------
# 7. Final audit
# ---------------------------------------------------------------

audit_628hc5a2r2 = pd.DataFrame(
    {
        "check": [
            "frozen_architecture_uncertainty_only",
            "frozen_policy_exact",
            "frozen_nominal_coverage_080",
            "frozen_threshold_exact",
            "official_test_label_marker_false",
            "all_final_risk_shapes_259",
            "all_final_risk_values_finite",
            "all_final_risk_values_within_0_1",
            "historical_cluster_array_exact",
            "historical_cluster_hash_exact",
            "reproduction_allclose_atol_1e14",
            "reproduction_sorted_order_exact",
            "reproduction_zero_threshold_disagreements",
            "canonical_object_shape_259",
            "canonical_object_exact_vs_FCR2",
            "no_test_file_read",
            "no_test_label_read",
            "no_model_execution",
            "no_test_scoring",
            "no_threshold_tuning",
            "no_files_written",
        ],

        "passed": [
            frozen_architecture_628hc5a2r2
            ==
            "uncertainty_only",

            frozen_policy_628hc5a2r2
            ==
            "uncertainty_only_fixed_threshold",

            frozen_coverage_628hc5a2r2
            ==
            0.80,

            frozen_threshold_628hc5a2r2
            ==
            0.8003345892330226,

            (
                "FD002_OFFICIAL_TEST_LABELS_READ_628A"
                in globals()
                and
                bool(
                    FD002_OFFICIAL_TEST_LABELS_READ_628A
                )
                is False
            ),

            shape_contract_628hc5a2r2,

            finite_contract_628hc5a2r2,

            range_contract_628hc5a2r2,

            historical_cluster_exact_628hc5a2r2,

            historical_hash_cluster_exact_628hc5a2r2,

            reproduction_close_628hc5a2r2,

            reproduction_order_exact_628hc5a2r2,

            reproduction_decision_equivalent_628hc5a2r2,

            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2.shape
            ==
            (259,),

            np.array_equal(
                FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2,
                fcr2_628hc5a2r2,
            ),

            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A2-R2 AUDIT ---")
display(
    audit_628hc5a2r2
)

all_checks_passed_628hc5a2r2 = bool(
    audit_628hc5a2r2[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a2r2:

    raise RuntimeError(
        "C5-A2-R2 final canonical TEST-risk identity lock failed."
    )

# ---------------------------------------------------------------
# 8. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A2-R2 FINAL SUMMARY")
print("=" * 94)

print(
    "Canonical TEST reliability source :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2R2,
)

print(
    "Canonical TEST risk shape          :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2.shape,
)

print(
    "Canonical TEST risk min/median/mean/max :",
    float(
        np.min(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2
        )
    ),
    float(
        np.median(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2
        )
    ),
    float(
        np.mean(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2
        )
    ),
    float(
        np.max(
            FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2
        )
    ),
)

print(
    "Historical exact cluster          :",
    historical_cluster_exact_628hc5a2r2,
)

print(
    "Historical hash cluster exact     :",
    historical_hash_cluster_exact_628hc5a2r2,
)

print(
    "Reproduction tolerance            :",
    reproduction_atol_628hc5a2r2,
)

print(
    "Reproduction max abs difference   :",
    repr(
        reproduction_max_abs_diff_628hc5a2r2
    ),
)

print(
    "Reproduction sorted order exact   :",
    reproduction_order_exact_628hc5a2r2,
)

print(
    "Frozen-threshold disagreements    :",
    reproduction_threshold_disagreements_628hc5a2r2,
)

print(
    "Canonical SHA-256                 :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2R2,
)

print(
    "All checks passed                 :",
    all_checks_passed_628hc5a2r2,
)

print("Official TEST labels read          : False")
print("TEST scored                        : False")
print("Threshold tuned or modified        : False")
print("Operational TEST mask frozen       : False")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files written                      : False")

print("\n=== END STEP 6.28H-C5-A2-R2 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A3
# APPLY FROZEN FD002 ABSTENTION POLICY TO CANONICAL TEST RISK
# ================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A3 — APPLY FROZEN FD002 ABSTENTION POLICY TO CANONICAL TEST RISK")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconfirm frozen pre-TEST contract
# ---------------------------------------------------------------

frozen_architecture_628hc5a3 = str(
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B[
        "reliability_architecture"
    ]
)

frozen_policy_628hc5a3 = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

frozen_nominal_coverage_628hc5a3 = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

frozen_threshold_628hc5a3 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

if frozen_architecture_628hc5a3 != "uncertainty_only":
    raise RuntimeError(
        "Frozen reliability architecture changed unexpectedly."
    )

if frozen_policy_628hc5a3 != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Frozen operating policy changed unexpectedly."
    )

if frozen_nominal_coverage_628hc5a3 != 0.80:
    raise RuntimeError(
        "Frozen nominal coverage changed unexpectedly."
    )

if frozen_threshold_628hc5a3 != 0.8003345892330226:
    raise RuntimeError(
        "Frozen risk threshold changed unexpectedly."
    )

# ---------------------------------------------------------------
# 2. TEST-label safety barrier
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Official TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is True. Stop."
    )

# ---------------------------------------------------------------
# 3. Load canonical TEST reliability-risk vector
# ---------------------------------------------------------------

canonical_test_risk_628hc5a3 = np.asarray(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2,
    dtype=np.float64,
).reshape(-1)

if canonical_test_risk_628hc5a3.shape != (259,):
    raise RuntimeError(
        "Canonical TEST risk does not have shape (259,)."
    )

if not np.isfinite(canonical_test_risk_628hc5a3).all():
    raise RuntimeError(
        "Canonical TEST risk contains non-finite values."
    )

if (
    np.min(canonical_test_risk_628hc5a3) < 0.0
    or
    np.max(canonical_test_risk_628hc5a3) > 1.0
):
    raise RuntimeError(
        "Canonical TEST risk falls outside [0,1]."
    )

# ---------------------------------------------------------------
# 4. Reverify canonical risk hash
# ---------------------------------------------------------------

def sha256_array_628hc5a3(arr):
    contiguous_628hc5a3 = np.ascontiguousarray(arr)

    return hashlib.sha256(
        contiguous_628hc5a3.tobytes(order="C")
    ).hexdigest()


canonical_risk_hash_628hc5a3 = sha256_array_628hc5a3(
    np.ascontiguousarray(
        canonical_test_risk_628hc5a3,
        dtype=np.float64,
    )
)

expected_canonical_hash_628hc5a3 = str(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2R2
)

canonical_hash_exact_628hc5a3 = bool(
    canonical_risk_hash_628hc5a3
    ==
    expected_canonical_hash_628hc5a3
)

if not canonical_hash_exact_628hc5a3:
    raise RuntimeError(
        "Canonical TEST risk hash changed after C5-A2-R2."
    )

# ---------------------------------------------------------------
# 5. Apply already-frozen C4 threshold
#
# Lower reliability risk is accepted.
#
# ACCEPT <=> risk <= frozen threshold
#
# No threshold is estimated, tuned, or modified here.
# ---------------------------------------------------------------

accept_mask_628hc5a3 = (
    canonical_test_risk_628hc5a3
    <=
    frozen_threshold_628hc5a3
)

abstain_mask_628hc5a3 = np.logical_not(
    accept_mask_628hc5a3
)

accepted_count_628hc5a3 = int(
    np.count_nonzero(
        accept_mask_628hc5a3
    )
)

abstained_count_628hc5a3 = int(
    np.count_nonzero(
        abstain_mask_628hc5a3
    )
)

realized_coverage_628hc5a3 = float(
    np.mean(
        accept_mask_628hc5a3
    )
)

realized_abstention_628hc5a3 = float(
    np.mean(
        abstain_mask_628hc5a3
    )
)

# ---------------------------------------------------------------
# 6. Threshold-boundary diagnostic
# ---------------------------------------------------------------

threshold_margin_628hc5a3 = (
    canonical_test_risk_628hc5a3
    -
    frozen_threshold_628hc5a3
)

exact_threshold_ties_628hc5a3 = int(
    np.count_nonzero(
        canonical_test_risk_628hc5a3
        ==
        frozen_threshold_628hc5a3
    )
)

nearest_abs_threshold_distance_628hc5a3 = float(
    np.min(
        np.abs(
            threshold_margin_628hc5a3
        )
    )
)

# ---------------------------------------------------------------
# 7. Freeze operational decisions in memory
# ---------------------------------------------------------------

FD002_TEST_ACCEPT_MASK_628HC5A3 = (
    accept_mask_628hc5a3.copy()
)

FD002_TEST_ABSTAIN_MASK_628HC5A3 = (
    abstain_mask_628hc5a3.copy()
)

FD002_TEST_REALIZED_COVERAGE_628HC5A3 = (
    realized_coverage_628hc5a3
)

FD002_TEST_REALIZED_ABSTENTION_628HC5A3 = (
    realized_abstention_628hc5a3
)

FD002_TEST_ACCEPTED_COUNT_628HC5A3 = (
    accepted_count_628hc5a3
)

FD002_TEST_ABSTAINED_COUNT_628HC5A3 = (
    abstained_count_628hc5a3
)

# ---------------------------------------------------------------
# 8. Decision hashes
# ---------------------------------------------------------------

accept_hash_628hc5a3 = sha256_array_628hc5a3(
    np.ascontiguousarray(
        FD002_TEST_ACCEPT_MASK_628HC5A3,
        dtype=np.bool_,
    )
)

abstain_hash_628hc5a3 = sha256_array_628hc5a3(
    np.ascontiguousarray(
        FD002_TEST_ABSTAIN_MASK_628HC5A3,
        dtype=np.bool_,
    )
)

FD002_TEST_ACCEPT_MASK_SHA256_628HC5A3 = (
    accept_hash_628hc5a3
)

FD002_TEST_ABSTAIN_MASK_SHA256_628HC5A3 = (
    abstain_hash_628hc5a3
)

# ---------------------------------------------------------------
# 9. Build label-free operational decision table
# ---------------------------------------------------------------

endpoint_position_628hc5a3 = np.arange(
    1,
    260,
    dtype=np.int64,
)

FD002_TEST_OPERATING_DECISIONS_628HC5A3 = pd.DataFrame(
    {
        "endpoint_position_1based":
            endpoint_position_628hc5a3,

        "reliability_risk":
            canonical_test_risk_628hc5a3,

        "frozen_risk_threshold":
            np.full(
                259,
                frozen_threshold_628hc5a3,
                dtype=np.float64,
            ),

        "accepted":
            FD002_TEST_ACCEPT_MASK_628HC5A3,

        "abstained":
            FD002_TEST_ABSTAIN_MASK_628HC5A3,

        "risk_minus_threshold":
            threshold_margin_628hc5a3,
    }
)

print("\n--- FROZEN TEST OPERATING-POLICY APPLICATION ---")

print(
    "Reliability architecture :",
    frozen_architecture_628hc5a3,
)

print(
    "Operating policy         :",
    frozen_policy_628hc5a3,
)

print(
    "Nominal DEV coverage     :",
    frozen_nominal_coverage_628hc5a3,
)

print(
    "Frozen risk threshold    :",
    repr(
        frozen_threshold_628hc5a3
    ),
)

print(
    "TEST accepted endpoints  :",
    accepted_count_628hc5a3,
)

print(
    "TEST abstained endpoints :",
    abstained_count_628hc5a3,
)

print(
    "TEST realized coverage   :",
    repr(
        realized_coverage_628hc5a3
    ),
)

print(
    "TEST realized abstention :",
    repr(
        realized_abstention_628hc5a3
    ),
)

print(
    "Exact threshold ties     :",
    exact_threshold_ties_628hc5a3,
)

print(
    "Nearest threshold distance:",
    repr(
        nearest_abs_threshold_distance_628hc5a3
    ),
)

print("\n--- FIRST 15 LABEL-FREE TEST DECISIONS ---")

display(
    FD002_TEST_OPERATING_DECISIONS_628HC5A3.head(
        15
    )
)

# ---------------------------------------------------------------
# 10. In-memory policy-application metadata
# ---------------------------------------------------------------

FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3 = {
    "dataset":
        "FD002",

    "population":
        "all_259_engines",

    "reliability_architecture":
        frozen_architecture_628hc5a3,

    "operating_policy":
        frozen_policy_628hc5a3,

    "canonical_test_risk_source":
        FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2R2,

    "canonical_test_risk_sha256":
        canonical_risk_hash_628hc5a3,

    "nominal_coverage":
        frozen_nominal_coverage_628hc5a3,

    "frozen_risk_threshold":
        frozen_threshold_628hc5a3,

    "decision_rule":
        "accept iff reliability_risk <= frozen_risk_threshold",

    "accepted_count":
        accepted_count_628hc5a3,

    "abstained_count":
        abstained_count_628hc5a3,

    "realized_coverage":
        realized_coverage_628hc5a3,

    "realized_abstention":
        realized_abstention_628hc5a3,

    "accept_mask_sha256":
        accept_hash_628hc5a3,

    "abstain_mask_sha256":
        abstain_hash_628hc5a3,

    "official_test_labels_read":
        False,

    "test_scored":
        False,
}

# ---------------------------------------------------------------
# 11. Audit
# ---------------------------------------------------------------

audit_628hc5a3 = pd.DataFrame(
    {
        "check": [
            "frozen_architecture_uncertainty_only",
            "frozen_policy_exact",
            "frozen_nominal_coverage_080",
            "frozen_threshold_exact",
            "official_test_label_marker_false",
            "canonical_risk_shape_259",
            "canonical_risk_hash_exact",
            "accept_mask_shape_259",
            "abstain_mask_shape_259",
            "masks_boolean",
            "accept_abstain_complements",
            "decision_count_sums_to_259",
            "realized_coverage_exact_from_mask",
            "realized_abstention_exact_from_mask",
            "decision_table_rows_259",
            "no_test_file_read",
            "no_test_label_read",
            "no_test_scoring",
            "no_threshold_tuning",
            "no_threshold_modification",
            "no_model_execution",
            "no_model_retraining",
            "no_files_written",
        ],

        "passed": [
            frozen_architecture_628hc5a3
            ==
            "uncertainty_only",

            frozen_policy_628hc5a3
            ==
            "uncertainty_only_fixed_threshold",

            frozen_nominal_coverage_628hc5a3
            ==
            0.80,

            frozen_threshold_628hc5a3
            ==
            0.8003345892330226,

            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            )
            is False,

            canonical_test_risk_628hc5a3.shape
            ==
            (259,),

            canonical_hash_exact_628hc5a3,

            FD002_TEST_ACCEPT_MASK_628HC5A3.shape
            ==
            (259,),

            FD002_TEST_ABSTAIN_MASK_628HC5A3.shape
            ==
            (259,),

            (
                FD002_TEST_ACCEPT_MASK_628HC5A3.dtype
                ==
                np.bool_
                and
                FD002_TEST_ABSTAIN_MASK_628HC5A3.dtype
                ==
                np.bool_
            ),

            bool(
                np.array_equal(
                    FD002_TEST_ABSTAIN_MASK_628HC5A3,
                    np.logical_not(
                        FD002_TEST_ACCEPT_MASK_628HC5A3
                    ),
                )
            ),

            (
                accepted_count_628hc5a3
                +
                abstained_count_628hc5a3
                ==
                259
            ),

            realized_coverage_628hc5a3
            ==
            float(
                np.mean(
                    FD002_TEST_ACCEPT_MASK_628HC5A3
                )
            ),

            realized_abstention_628hc5a3
            ==
            float(
                np.mean(
                    FD002_TEST_ABSTAIN_MASK_628HC5A3
                )
            ),

            len(
                FD002_TEST_OPERATING_DECISIONS_628HC5A3
            )
            ==
            259,

            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A3 AUDIT ---")

display(
    audit_628hc5a3
)

all_checks_passed_628hc5a3 = bool(
    audit_628hc5a3[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a3:
    raise RuntimeError(
        "C5-A3 frozen TEST operating-policy application failed."
    )

# ---------------------------------------------------------------
# 12. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A3 FINAL SUMMARY")
print("=" * 94)

print(
    "Reliability architecture       :",
    frozen_architecture_628hc5a3,
)

print(
    "Operating policy               :",
    frozen_policy_628hc5a3,
)

print(
    "Canonical TEST risk source     :",
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2R2,
)

print(
    "Frozen exact risk threshold    :",
    repr(
        frozen_threshold_628hc5a3
    ),
)

print(
    "Accepted TEST endpoints        :",
    accepted_count_628hc5a3,
)

print(
    "Abstained TEST endpoints       :",
    abstained_count_628hc5a3,
)

print(
    "Realized TEST coverage         :",
    repr(
        realized_coverage_628hc5a3
    ),
)

print(
    "Realized TEST abstention       :",
    repr(
        realized_abstention_628hc5a3
    ),
)

print(
    "Accept-mask SHA-256            :",
    FD002_TEST_ACCEPT_MASK_SHA256_628HC5A3,
)

print(
    "Abstain-mask SHA-256           :",
    FD002_TEST_ABSTAIN_MASK_SHA256_628HC5A3,
)

print(
    "All checks passed              :",
    all_checks_passed_628hc5a3,
)

print("Official TEST labels read       : False")
print("TEST scored                     : False")
print("Threshold tuned or modified     : False")
print("Model executed                  : False")
print("Model retrained                 : False")
print("Files written                   : False")

print("\n=== END STEP 6.28H-C5-A3 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-F1-A
# FREEZE FD002 TEST RELIABILITY RISK + OPERATING DECISIONS
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F1-A — FREEZE FD002 TEST RELIABILITY RISK + OPERATING DECISIONS")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconfirm the already-frozen development contract
# ---------------------------------------------------------------

architecture_628hc5f1a = str(
    FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B[
        "reliability_architecture"
    ]
)

policy_628hc5f1a = str(
    FD002_FROZEN_C4_SELECTED_POLICY_NAME_628HC4F1B
)

nominal_coverage_628hc5f1a = float(
    FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
)

threshold_628hc5f1a = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

if architecture_628hc5f1a != "uncertainty_only":
    raise RuntimeError("Frozen architecture mismatch.")

if policy_628hc5f1a != "uncertainty_only_fixed_threshold":
    raise RuntimeError("Frozen policy mismatch.")

if nominal_coverage_628hc5f1a != 0.80:
    raise RuntimeError("Frozen nominal coverage mismatch.")

if threshold_628hc5f1a != 0.8003345892330226:
    raise RuntimeError("Frozen threshold mismatch.")

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is True. Stop before writing artifacts."
    )

# ---------------------------------------------------------------
# 2. Canonical in-memory artifacts from C5-A2-R2 / C5-A3
# ---------------------------------------------------------------

risk_628hc5f1a = np.asarray(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2,
    dtype=np.float64,
).reshape(-1)

accept_628hc5f1a = np.asarray(
    FD002_TEST_ACCEPT_MASK_628HC5A3,
    dtype=np.bool_,
).reshape(-1)

abstain_628hc5f1a = np.asarray(
    FD002_TEST_ABSTAIN_MASK_628HC5A3,
    dtype=np.bool_,
).reshape(-1)

decision_table_628hc5f1a = (
    FD002_TEST_OPERATING_DECISIONS_628HC5A3.copy()
)

if risk_628hc5f1a.shape != (259,):
    raise RuntimeError("Canonical TEST risk shape mismatch.")

if accept_628hc5f1a.shape != (259,):
    raise RuntimeError("TEST accept-mask shape mismatch.")

if abstain_628hc5f1a.shape != (259,):
    raise RuntimeError("TEST abstain-mask shape mismatch.")

if len(decision_table_628hc5f1a) != 259:
    raise RuntimeError("TEST decision-table row count mismatch.")

if not np.array_equal(
    abstain_628hc5f1a,
    np.logical_not(accept_628hc5f1a),
):
    raise RuntimeError("Accept/abstain masks are not complements.")

# ---------------------------------------------------------------
# 3. Array hash helper
# ---------------------------------------------------------------

def sha256_array_628hc5f1a(arr):
    contiguous = np.ascontiguousarray(arr)
    return hashlib.sha256(
        contiguous.tobytes(order="C")
    ).hexdigest()


risk_hash_628hc5f1a = sha256_array_628hc5f1a(
    np.asarray(risk_628hc5f1a, dtype=np.float64)
)

accept_hash_628hc5f1a = sha256_array_628hc5f1a(
    np.asarray(accept_628hc5f1a, dtype=np.bool_)
)

abstain_hash_628hc5f1a = sha256_array_628hc5f1a(
    np.asarray(abstain_628hc5f1a, dtype=np.bool_)
)

if risk_hash_628hc5f1a != (
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2R2
):
    raise RuntimeError("Canonical TEST risk hash mismatch.")

if accept_hash_628hc5f1a != (
    FD002_TEST_ACCEPT_MASK_SHA256_628HC5A3
):
    raise RuntimeError("TEST accept-mask hash mismatch.")

if abstain_hash_628hc5f1a != (
    FD002_TEST_ABSTAIN_MASK_SHA256_628HC5A3
):
    raise RuntimeError("TEST abstain-mask hash mismatch.")

# ---------------------------------------------------------------
# 4. Freeze directory and filenames
# ---------------------------------------------------------------

project_root_628hc5f1a = Path(
    r"C:\RUL_Reliability_Research"
)

freeze_dir_628hc5f1a = (
    project_root_628hc5f1a
    / "06_results"
    / "step6_reliability_signals"
    / "fd002_c5"
)

freeze_dir_628hc5f1a.mkdir(
    parents=True,
    exist_ok=True,
)

npz_path_628hc5f1a = (
    freeze_dir_628hc5f1a
    / "fd002_c5_test_reliability_and_decisions.npz"
)

csv_path_628hc5f1a = (
    freeze_dir_628hc5f1a
    / "fd002_c5_test_operating_decisions.csv"
)

metadata_path_628hc5f1a = (
    freeze_dir_628hc5f1a
    / "fd002_c5_test_operating_decisions_metadata.json"
)

# ---------------------------------------------------------------
# 5. Write NPZ and CSV
# ---------------------------------------------------------------

np.savez_compressed(
    npz_path_628hc5f1a,
    canonical_test_risk=risk_628hc5f1a,
    accept_mask=accept_628hc5f1a,
    abstain_mask=abstain_628hc5f1a,
    frozen_risk_threshold=np.array(
        threshold_628hc5f1a,
        dtype=np.float64,
    ),
    nominal_coverage=np.array(
        nominal_coverage_628hc5f1a,
        dtype=np.float64,
    ),
    realized_coverage=np.array(
        FD002_TEST_REALIZED_COVERAGE_628HC5A3,
        dtype=np.float64,
    ),
    realized_abstention=np.array(
        FD002_TEST_REALIZED_ABSTENTION_628HC5A3,
        dtype=np.float64,
    ),
    accepted_count=np.array(
        FD002_TEST_ACCEPTED_COUNT_628HC5A3,
        dtype=np.int64,
    ),
    abstained_count=np.array(
        FD002_TEST_ABSTAINED_COUNT_628HC5A3,
        dtype=np.int64,
    ),
)

decision_table_628hc5f1a.to_csv(
    csv_path_628hc5f1a,
    index=False,
)

# ---------------------------------------------------------------
# 6. File hash helper and metadata write
# ---------------------------------------------------------------

def sha256_file_628hc5f1a(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


npz_file_hash_628hc5f1a = sha256_file_628hc5f1a(
    npz_path_628hc5f1a
)

csv_file_hash_628hc5f1a = sha256_file_628hc5f1a(
    csv_path_628hc5f1a
)

metadata_628hc5f1a = {
    "step":
        "6.28H-C5-F1-A",

    "dataset":
        "FD002",

    "population":
        "all_259_engines",

    "reliability_architecture":
        architecture_628hc5f1a,

    "operating_policy":
        policy_628hc5f1a,

    "nominal_coverage":
        nominal_coverage_628hc5f1a,

    "frozen_risk_threshold":
        threshold_628hc5f1a,

    "canonical_test_risk_source":
        FD002_CANONICAL_TEST_UNCERTAINTY_RISK_SOURCE_628HC5A2R2,

    "canonical_test_risk_sha256":
        risk_hash_628hc5f1a,

    "accept_mask_sha256":
        accept_hash_628hc5f1a,

    "abstain_mask_sha256":
        abstain_hash_628hc5f1a,

    "accepted_count":
        int(FD002_TEST_ACCEPTED_COUNT_628HC5A3),

    "abstained_count":
        int(FD002_TEST_ABSTAINED_COUNT_628HC5A3),

    "realized_coverage":
        float(FD002_TEST_REALIZED_COVERAGE_628HC5A3),

    "realized_abstention":
        float(FD002_TEST_REALIZED_ABSTENTION_628HC5A3),

    "decision_rule":
        "accept iff reliability_risk <= frozen_risk_threshold",

    "npz_filename":
        npz_path_628hc5f1a.name,

    "npz_file_sha256":
        npz_file_hash_628hc5f1a,

    "decision_csv_filename":
        csv_path_628hc5f1a.name,

    "decision_csv_sha256":
        csv_file_hash_628hc5f1a,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "threshold_tuned_or_modified":
        False,

    "model_executed":
        False,

    "model_retrained":
        False,
}

metadata_path_628hc5f1a.write_text(
    json.dumps(
        metadata_628hc5f1a,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

metadata_file_hash_628hc5f1a = sha256_file_628hc5f1a(
    metadata_path_628hc5f1a
)

# ---------------------------------------------------------------
# 7. Expose frozen artifact paths/hashes for F1-B
# ---------------------------------------------------------------

FD002_C5_F1A_NPZ_PATH = npz_path_628hc5f1a
FD002_C5_F1A_DECISION_CSV_PATH = csv_path_628hc5f1a
FD002_C5_F1A_METADATA_PATH = metadata_path_628hc5f1a

FD002_C5_F1A_NPZ_SHA256 = npz_file_hash_628hc5f1a
FD002_C5_F1A_DECISION_CSV_SHA256 = csv_file_hash_628hc5f1a
FD002_C5_F1A_METADATA_SHA256 = metadata_file_hash_628hc5f1a

# ---------------------------------------------------------------
# 8. F1-A write audit
# ---------------------------------------------------------------

all_files_written_628hc5f1a = bool(
    npz_path_628hc5f1a.exists()
    and
    csv_path_628hc5f1a.exists()
    and
    metadata_path_628hc5f1a.exists()
    and
    npz_path_628hc5f1a.stat().st_size > 0
    and
    csv_path_628hc5f1a.stat().st_size > 0
    and
    metadata_path_628hc5f1a.stat().st_size > 0
)

print("\n--- FREEZE DIRECTORY ---")
print(freeze_dir_628hc5f1a)

print("\n--- WRITTEN ARTIFACTS ---")
print(
    npz_path_628hc5f1a.name,
    "| exists =",
    npz_path_628hc5f1a.exists(),
    "| bytes =",
    npz_path_628hc5f1a.stat().st_size,
)

print(
    csv_path_628hc5f1a.name,
    "| exists =",
    csv_path_628hc5f1a.exists(),
    "| bytes =",
    csv_path_628hc5f1a.stat().st_size,
)

print(
    metadata_path_628hc5f1a.name,
    "| exists =",
    metadata_path_628hc5f1a.exists(),
    "| bytes =",
    metadata_path_628hc5f1a.stat().st_size,
)

print("\n--- ARRAY SHA-256 ---")
print("canonical TEST risk :", risk_hash_628hc5f1a)
print("accept mask         :", accept_hash_628hc5f1a)
print("abstain mask        :", abstain_hash_628hc5f1a)

print("\n--- FILE SHA-256 ---")
print("NPZ      :", npz_file_hash_628hc5f1a)
print("CSV      :", csv_file_hash_628hc5f1a)
print("metadata :", metadata_file_hash_628hc5f1a)

print("\n--- F1-A STATUS ---")
print("All files written correctly     :", all_files_written_628hc5f1a)
print("Independent reload verified     :", False)
print("Official TEST labels read       :", False)
print("TEST scored                     :", False)
print("Threshold tuned or modified     :", False)
print("Model executed                  :", False)
print("Model retrained                 :", False)
print("Files written                   :", True)

if not all_files_written_628hc5f1a:
    raise RuntimeError(
        "C5-F1-A artifact freeze did not complete correctly."
    )

print(
    "\nNext required step: "
    "C5-F1-B independent reload and exact verification."
)

print("\n=== END STEP 6.28H-C5-F1-A ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-F1-B
# INDEPENDENT RELOAD + EXACT VERIFICATION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F1-B — INDEPENDENT RELOAD + EXACT VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Resolve F1-A artifacts and enforce TEST-label barrier
# ---------------------------------------------------------------

npz_path = Path(FD002_C5_F1A_NPZ_PATH)
csv_path = Path(FD002_C5_F1A_DECISION_CSV_PATH)
metadata_path = Path(FD002_C5_F1A_METADATA_PATH)

if not (npz_path.exists() and csv_path.exists() and metadata_path.exists()):
    raise RuntimeError(
        "One or more C5-F1-A frozen artifacts are missing."
    )

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Official TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is True. Stop."
    )

# ---------------------------------------------------------------
# 2. Verify frozen file hashes before loading content
# ---------------------------------------------------------------

def sha256_file_628hc5f1b(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


npz_file_hash = sha256_file_628hc5f1b(
    npz_path
)

csv_file_hash = sha256_file_628hc5f1b(
    csv_path
)

metadata_file_hash = sha256_file_628hc5f1b(
    metadata_path
)

npz_file_hash_exact = (
    npz_file_hash
    ==
    FD002_C5_F1A_NPZ_SHA256
)

csv_file_hash_exact = (
    csv_file_hash
    ==
    FD002_C5_F1A_DECISION_CSV_SHA256
)

metadata_file_hash_exact = (
    metadata_file_hash
    ==
    FD002_C5_F1A_METADATA_SHA256
)

# ---------------------------------------------------------------
# 3. Independently reload NPZ
# ---------------------------------------------------------------

z = np.load(
    npz_path,
    allow_pickle=False,
)

expected_npz_keys = {
    "canonical_test_risk",
    "accept_mask",
    "abstain_mask",
    "frozen_risk_threshold",
    "nominal_coverage",
    "realized_coverage",
    "realized_abstention",
    "accepted_count",
    "abstained_count",
}

npz_keys_exact = (
    set(z.files)
    ==
    expected_npz_keys
)

risk_r = np.asarray(
    z["canonical_test_risk"],
    dtype=np.float64,
).reshape(-1)

accept_r = np.asarray(
    z["accept_mask"],
    dtype=np.bool_,
).reshape(-1)

abstain_r = np.asarray(
    z["abstain_mask"],
    dtype=np.bool_,
).reshape(-1)

threshold_r = float(
    z["frozen_risk_threshold"]
)

nominal_coverage_r = float(
    z["nominal_coverage"]
)

realized_coverage_r = float(
    z["realized_coverage"]
)

realized_abstention_r = float(
    z["realized_abstention"]
)

accepted_count_r = int(
    z["accepted_count"]
)

abstained_count_r = int(
    z["abstained_count"]
)

z.close()

# ---------------------------------------------------------------
# 4. Exact NPZ-vs-memory verification
# ---------------------------------------------------------------

risk_m = np.asarray(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2R2,
    dtype=np.float64,
).reshape(-1)

accept_m = np.asarray(
    FD002_TEST_ACCEPT_MASK_628HC5A3,
    dtype=np.bool_,
).reshape(-1)

abstain_m = np.asarray(
    FD002_TEST_ABSTAIN_MASK_628HC5A3,
    dtype=np.bool_,
).reshape(-1)

risk_exact_vs_memory = np.array_equal(
    risk_r,
    risk_m,
)

accept_exact_vs_memory = np.array_equal(
    accept_r,
    accept_m,
)

abstain_exact_vs_memory = np.array_equal(
    abstain_r,
    abstain_m,
)

def sha256_array_628hc5f1b(arr):
    a = np.ascontiguousarray(
        arr
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


risk_hash_r = sha256_array_628hc5f1b(
    np.asarray(
        risk_r,
        dtype=np.float64,
    )
)

accept_hash_r = sha256_array_628hc5f1b(
    np.asarray(
        accept_r,
        dtype=np.bool_,
    )
)

abstain_hash_r = sha256_array_628hc5f1b(
    np.asarray(
        abstain_r,
        dtype=np.bool_,
    )
)

risk_array_hash_exact = (
    risk_hash_r
    ==
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_HASH_628HC5A2R2
)

accept_array_hash_exact = (
    accept_hash_r
    ==
    FD002_TEST_ACCEPT_MASK_SHA256_628HC5A3
)

abstain_array_hash_exact = (
    abstain_hash_r
    ==
    FD002_TEST_ABSTAIN_MASK_SHA256_628HC5A3
)

scalar_contract_exact = bool(
    threshold_r
    ==
    float(
        FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
    )
    and
    nominal_coverage_r
    ==
    float(
        FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_628HC4F1B
    )
    and
    accepted_count_r
    ==
    int(
        FD002_TEST_ACCEPTED_COUNT_628HC5A3
    )
    and
    abstained_count_r
    ==
    int(
        FD002_TEST_ABSTAINED_COUNT_628HC5A3
    )
    and
    realized_coverage_r
    ==
    float(
        FD002_TEST_REALIZED_COVERAGE_628HC5A3
    )
    and
    realized_abstention_r
    ==
    float(
        FD002_TEST_REALIZED_ABSTENTION_628HC5A3
    )
)

mask_complement_exact = np.array_equal(
    abstain_r,
    np.logical_not(
        accept_r
    ),
)

threshold_rule_exact = np.array_equal(
    accept_r,
    risk_r <= threshold_r,
)

# ---------------------------------------------------------------
# 5. Independently reload CSV
# ---------------------------------------------------------------

csv_r = pd.read_csv(
    csv_path
)

expected_csv_columns = [
    "endpoint_position_1based",
    "reliability_risk",
    "frozen_risk_threshold",
    "accepted",
    "abstained",
    "risk_minus_threshold",
]

csv_columns_exact = (
    list(csv_r.columns)
    ==
    expected_csv_columns
)

csv_rows_259 = (
    len(csv_r)
    ==
    259
)

csv_endpoint_order_exact = np.array_equal(
    csv_r[
        "endpoint_position_1based"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        260,
        dtype=np.int64,
    ),
)

csv_risk_close = np.allclose(
    csv_r[
        "reliability_risk"
    ].to_numpy(
        dtype=np.float64
    ),
    risk_r,
    rtol=0.0,
    atol=1e-15,
)

csv_threshold_exact = np.all(
    csv_r[
        "frozen_risk_threshold"
    ].to_numpy(
        dtype=np.float64
    )
    ==
    threshold_r
)

csv_accept_exact = np.array_equal(
    csv_r[
        "accepted"
    ].to_numpy(
        dtype=np.bool_
    ),
    accept_r,
)

csv_abstain_exact = np.array_equal(
    csv_r[
        "abstained"
    ].to_numpy(
        dtype=np.bool_
    ),
    abstain_r,
)

csv_margin_close = np.allclose(
    csv_r[
        "risk_minus_threshold"
    ].to_numpy(
        dtype=np.float64
    ),
    risk_r - threshold_r,
    rtol=0.0,
    atol=1e-15,
)

# ---------------------------------------------------------------
# 6. Independently reload metadata JSON
# ---------------------------------------------------------------

metadata_r = json.loads(
    metadata_path.read_text(
        encoding="utf-8"
    )
)

metadata_contract_exact = bool(
    metadata_r["step"]
    ==
    "6.28H-C5-F1-A"

    and

    metadata_r["dataset"]
    ==
    "FD002"

    and

    metadata_r["population"]
    ==
    "all_259_engines"

    and

    metadata_r[
        "reliability_architecture"
    ]
    ==
    "uncertainty_only"

    and

    metadata_r[
        "operating_policy"
    ]
    ==
    "uncertainty_only_fixed_threshold"

    and

    float(
        metadata_r[
            "nominal_coverage"
        ]
    )
    ==
    nominal_coverage_r

    and

    float(
        metadata_r[
            "frozen_risk_threshold"
        ]
    )
    ==
    threshold_r

    and

    int(
        metadata_r[
            "accepted_count"
        ]
    )
    ==
    accepted_count_r

    and

    int(
        metadata_r[
            "abstained_count"
        ]
    )
    ==
    abstained_count_r

    and

    float(
        metadata_r[
            "realized_coverage"
        ]
    )
    ==
    realized_coverage_r

    and

    float(
        metadata_r[
            "realized_abstention"
        ]
    )
    ==
    realized_abstention_r

    and

    metadata_r[
        "canonical_test_risk_sha256"
    ]
    ==
    risk_hash_r

    and

    metadata_r[
        "accept_mask_sha256"
    ]
    ==
    accept_hash_r

    and

    metadata_r[
        "abstain_mask_sha256"
    ]
    ==
    abstain_hash_r

    and

    metadata_r[
        "npz_file_sha256"
    ]
    ==
    npz_file_hash

    and

    metadata_r[
        "decision_csv_sha256"
    ]
    ==
    csv_file_hash

    and

    metadata_r[
        "official_test_labels_read"
    ]
    is False

    and

    metadata_r[
        "test_scored"
    ]
    is False

    and

    metadata_r[
        "threshold_tuned_or_modified"
    ]
    is False

    and

    metadata_r[
        "model_executed"
    ]
    is False

    and

    metadata_r[
        "model_retrained"
    ]
    is False
)

# ---------------------------------------------------------------
# 7. Audit
# ---------------------------------------------------------------

audit_628hc5f1b = pd.DataFrame(
    {
        "check": [
            "all_three_F1A_files_present",
            "npz_file_hash_exact",
            "decision_csv_file_hash_exact",
            "metadata_file_hash_exact",
            "npz_keys_exact",
            "risk_shape_259",
            "accept_shape_259",
            "abstain_shape_259",
            "risk_exact_vs_memory",
            "accept_exact_vs_memory",
            "abstain_exact_vs_memory",
            "risk_array_hash_exact",
            "accept_array_hash_exact",
            "abstain_array_hash_exact",
            "scalar_contract_exact",
            "mask_complement_exact",
            "threshold_rule_exact",
            "csv_columns_exact",
            "csv_rows_259",
            "csv_endpoint_order_exact",
            "csv_risk_roundtrip_close_1e15",
            "csv_threshold_exact",
            "csv_accept_exact",
            "csv_abstain_exact",
            "csv_margin_roundtrip_close_1e15",
            "metadata_contract_exact",
            "official_test_label_marker_false",
            "no_test_file_read",
            "no_test_label_read",
            "no_test_scoring",
            "no_threshold_tuning",
            "no_threshold_modification",
            "no_model_execution",
            "no_model_retraining",
            "no_files_modified",
        ],

        "passed": [
            (
                npz_path.exists()
                and
                csv_path.exists()
                and
                metadata_path.exists()
            ),

            npz_file_hash_exact,
            csv_file_hash_exact,
            metadata_file_hash_exact,
            npz_keys_exact,

            risk_r.shape == (259,),
            accept_r.shape == (259,),
            abstain_r.shape == (259,),

            risk_exact_vs_memory,
            accept_exact_vs_memory,
            abstain_exact_vs_memory,

            risk_array_hash_exact,
            accept_array_hash_exact,
            abstain_array_hash_exact,

            scalar_contract_exact,
            mask_complement_exact,
            threshold_rule_exact,

            csv_columns_exact,
            csv_rows_259,
            csv_endpoint_order_exact,
            csv_risk_close,
            csv_threshold_exact,
            csv_accept_exact,
            csv_abstain_exact,
            csv_margin_close,

            metadata_contract_exact,

            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            )
            is False,

            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print(
    "\n--- C5-F1-B INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc5f1b
)

all_checks_passed_628hc5f1b = bool(
    audit_628hc5f1b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5f1b:
    raise RuntimeError(
        "C5-F1-B independent reload verification failed."
    )

# ---------------------------------------------------------------
# 8. Preserve verified reload objects in memory
# ---------------------------------------------------------------

FD002_C5_RELOADED_CANONICAL_TEST_RISK_628HC5F1B = (
    risk_r.copy()
)

FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B = (
    accept_r.copy()
)

FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B = (
    abstain_r.copy()
)

FD002_C5_RELOADED_DECISION_TABLE_628HC5F1B = (
    csv_r.copy()
)

FD002_C5_RELOADED_METADATA_628HC5F1B = dict(
    metadata_r
)

# ---------------------------------------------------------------
# 9. Final summary
# ---------------------------------------------------------------

print(
    "\n"
    +
    "=" * 94
)

print(
    "STEP 6.28H-C5-F1-B FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Canonical TEST risk shape       :",
    risk_r.shape,
)

print(
    "Frozen exact threshold          :",
    repr(
        threshold_r
    ),
)

print(
    "Accepted TEST endpoints         :",
    accepted_count_r,
)

print(
    "Abstained TEST endpoints        :",
    abstained_count_r,
)

print(
    "Realized TEST coverage          :",
    repr(
        realized_coverage_r
    ),
)

print(
    "Risk array exact vs memory      :",
    risk_exact_vs_memory,
)

print(
    "Accept mask exact vs memory     :",
    accept_exact_vs_memory,
)

print(
    "Abstain mask exact vs memory    :",
    abstain_exact_vs_memory,
)

print(
    "All file hashes exact           :",
    bool(
        npz_file_hash_exact
        and
        csv_file_hash_exact
        and
        metadata_file_hash_exact
    ),
)

print(
    "Metadata contract exact         :",
    metadata_contract_exact,
)

print(
    "Independent reload verified     :",
    all_checks_passed_628hc5f1b,
)

print("Official TEST labels read       : False")
print("TEST scored                     : False")
print("Threshold tuned or modified     : False")
print("Model executed                  : False")
print("Model retrained                 : False")
print("Files modified                  : False")

print(
    "\n=== END STEP 6.28H-C5-F1-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-A4
# PRE-SCORING FD002 TEST PREDICTION LINEAGE AUDIT
# ================================================================

import hashlib
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 94)
print("STEP 6.28H-C5-A4 — PRE-SCORING FD002 TEST PREDICTION LINEAGE AUDIT")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Safety barrier
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Official TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST-label marker is already True. Stop."
    )

# ---------------------------------------------------------------
# 2. Explicit historical prediction candidates
# ---------------------------------------------------------------

candidate_names_628hc5a4 = [
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
    "TEST_DET_628FBR1",
    "TEST_MC_MEAN_FD002_628FAR2",
    "test_prediction_628fb",
    "test_prediction_628hc0r2",
]

candidate_arrays_628hc5a4 = {}
inventory_rows_628hc5a4 = []

for name in candidate_names_628hc5a4:
    exists = name in globals()

    if not exists:
        inventory_rows_628hc5a4.append(
            {
                "name": name,
                "exists": False,
                "shape": None,
                "finite": False,
                "min": np.nan,
                "median": np.nan,
                "mean": np.nan,
                "max": np.nan,
            }
        )
        continue

    arr = np.asarray(
        globals()[name],
        dtype=np.float64,
    ).reshape(-1)

    candidate_arrays_628hc5a4[name] = arr

    inventory_rows_628hc5a4.append(
        {
            "name": name,
            "exists": True,
            "shape": arr.shape,
            "finite": bool(np.isfinite(arr).all()),
            "min": float(np.min(arr)),
            "median": float(np.median(arr)),
            "mean": float(np.mean(arr)),
            "max": float(np.max(arr)),
        }
    )

inventory_628hc5a4 = pd.DataFrame(
    inventory_rows_628hc5a4
)

print("\n--- HISTORICAL TEST PREDICTION CANDIDATES ---")
display(inventory_628hc5a4)

# ---------------------------------------------------------------
# 3. Pairwise numerical agreement
# ---------------------------------------------------------------

pair_rows_628hc5a4 = []

available_names_628hc5a4 = list(
    candidate_arrays_628hc5a4.keys()
)

for i in range(len(available_names_628hc5a4)):
    for j in range(i + 1, len(available_names_628hc5a4)):
        name_a = available_names_628hc5a4[i]
        name_b = available_names_628hc5a4[j]

        a = candidate_arrays_628hc5a4[name_a]
        b = candidate_arrays_628hc5a4[name_b]

        same_shape = (
            a.shape == b.shape
        )

        if same_shape:
            array_equal = np.array_equal(
                a,
                b,
            )

            max_abs_error = float(
                np.max(
                    np.abs(
                        a - b
                    )
                )
            )

            allclose_1e14 = bool(
                np.allclose(
                    a,
                    b,
                    rtol=0.0,
                    atol=1e-14,
                )
            )

            spearman = float(
                spearmanr(
                    a,
                    b,
                ).statistic
            )
        else:
            array_equal = False
            max_abs_error = np.nan
            allclose_1e14 = False
            spearman = np.nan

        pair_rows_628hc5a4.append(
            {
                "object_a": name_a,
                "object_b": name_b,
                "same_shape": same_shape,
                "array_equal": array_equal,
                "allclose_1e14": allclose_1e14,
                "max_abs_error": max_abs_error,
                "spearman": spearman,
            }
        )

pairwise_628hc5a4 = pd.DataFrame(
    pair_rows_628hc5a4
)

print("\n--- PAIRWISE PREDICTION AGREEMENT ---")
display(pairwise_628hc5a4)

# ---------------------------------------------------------------
# 4. Focused lineage comparisons
# ---------------------------------------------------------------

required_names_628hc5a4 = [
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
    "TEST_DET_628FBR1",
    "TEST_MC_MEAN_FD002_628FAR2",
    "test_prediction_628hc0r2",
]

required_present_628hc5a4 = all(
    name in candidate_arrays_628hc5a4
    for name in required_names_628hc5a4
)

if required_present_628hc5a4:
    det_far2 = candidate_arrays_628hc5a4[
        "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2"
    ]

    det_fbr1 = candidate_arrays_628hc5a4[
        "TEST_DET_628FBR1"
    ]

    mc_mean = candidate_arrays_628hc5a4[
        "TEST_MC_MEAN_FD002_628FAR2"
    ]

    latest_prediction = candidate_arrays_628hc5a4[
        "test_prediction_628hc0r2"
    ]

    det_cluster_exact_628hc5a4 = bool(
        np.array_equal(
            det_far2,
            det_fbr1,
        )
    )

    latest_matches_det_exact_628hc5a4 = bool(
        np.array_equal(
            latest_prediction,
            det_far2,
        )
    )

    latest_matches_mc_exact_628hc5a4 = bool(
        np.array_equal(
            latest_prediction,
            mc_mean,
        )
    )

    latest_matches_det_close_628hc5a4 = bool(
        np.allclose(
            latest_prediction,
            det_far2,
            rtol=0.0,
            atol=1e-14,
        )
    )

    latest_matches_mc_close_628hc5a4 = bool(
        np.allclose(
            latest_prediction,
            mc_mean,
            rtol=0.0,
            atol=1e-14,
        )
    )
else:
    det_cluster_exact_628hc5a4 = False
    latest_matches_det_exact_628hc5a4 = False
    latest_matches_mc_exact_628hc5a4 = False
    latest_matches_det_close_628hc5a4 = False
    latest_matches_mc_close_628hc5a4 = False

focused_628hc5a4 = pd.DataFrame(
    {
        "check": [
            "required_prediction_objects_present",
            "all_required_shapes_259",
            "all_required_values_finite",
            "deterministic_historical_cluster_exact",
            "latest_prediction_matches_deterministic_exact",
            "latest_prediction_matches_mc_mean_exact",
            "latest_prediction_matches_deterministic_close_1e14",
            "latest_prediction_matches_mc_mean_close_1e14",
        ],
        "value": [
            required_present_628hc5a4,
            (
                required_present_628hc5a4
                and
                all(
                    candidate_arrays_628hc5a4[name].shape == (259,)
                    for name in required_names_628hc5a4
                )
            ),
            (
                required_present_628hc5a4
                and
                all(
                    np.isfinite(
                        candidate_arrays_628hc5a4[name]
                    ).all()
                    for name in required_names_628hc5a4
                )
            ),
            det_cluster_exact_628hc5a4,
            latest_matches_det_exact_628hc5a4,
            latest_matches_mc_exact_628hc5a4,
            latest_matches_det_close_628hc5a4,
            latest_matches_mc_close_628hc5a4,
        ],
    }
)

print("\n--- FOCUSED PREDICTION LINEAGE AUDIT ---")
display(focused_628hc5a4)

# ---------------------------------------------------------------
# 5. Principal candidate hashes
# ---------------------------------------------------------------

def sha256_float64_array_628hc5a4(arr):
    a = np.ascontiguousarray(
        np.asarray(
            arr,
            dtype=np.float64,
        )
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


hash_rows_628hc5a4 = []

for name in required_names_628hc5a4:
    if name in candidate_arrays_628hc5a4:
        hash_rows_628hc5a4.append(
            {
                "name": name,
                "sha256": sha256_float64_array_628hc5a4(
                    candidate_arrays_628hc5a4[name]
                ),
            }
        )

hashes_628hc5a4 = pd.DataFrame(
    hash_rows_628hc5a4
)

print("\n--- PRINCIPAL PREDICTION SHA-256 ---")
display(hashes_628hc5a4)

# ---------------------------------------------------------------
# 6. Safety/audit summary
# ---------------------------------------------------------------

audit_628hc5a4 = pd.DataFrame(
    {
        "check": [
            "official_test_label_marker_false",
            "required_prediction_objects_present",
            "all_required_shapes_259",
            "all_required_values_finite",
            "no_test_file_read",
            "no_test_label_read",
            "no_test_scoring",
            "no_threshold_tuning",
            "no_threshold_modification",
            "no_model_execution",
            "no_model_retraining",
            "no_files_written",
        ],
        "passed": [
            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
            ) is False,
            required_present_628hc5a4,
            (
                required_present_628hc5a4
                and
                all(
                    candidate_arrays_628hc5a4[name].shape == (259,)
                    for name in required_names_628hc5a4
                )
            ),
            (
                required_present_628hc5a4
                and
                all(
                    np.isfinite(
                        candidate_arrays_628hc5a4[name]
                    ).all()
                    for name in required_names_628hc5a4
                )
            ),
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C5-A4 AUDIT ---")
display(audit_628hc5a4)

all_checks_passed_628hc5a4 = bool(
    audit_628hc5a4[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a4:
    raise RuntimeError(
        "C5-A4 prediction-lineage audit failed."
    )

print("\n" + "=" * 94)
print("STEP 6.28H-C5-A4 FINAL SUMMARY")
print("=" * 94)

print(
    "Prediction candidates inspected       :",
    len(
        available_names_628hc5a4
    ),
)

print(
    "Deterministic historical cluster exact:",
    det_cluster_exact_628hc5a4,
)

print(
    "Latest matches deterministic exactly  :",
    latest_matches_det_exact_628hc5a4,
)

print(
    "Latest matches MC mean exactly         :",
    latest_matches_mc_exact_628hc5a4,
)

print(
    "All audit checks passed                :",
    all_checks_passed_628hc5a4,
)

print("Official TEST labels read              : False")
print("TEST scored                            : False")
print("Prediction source selected             : False")
print("Threshold tuned or modified            : False")
print("Model executed                         : False")
print("Model retrained                        : False")
print("Files written                          : False")

print("\n=== END STEP 6.28H-C5-A4 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-F2-A
# FREEZE FD002 CANONICAL TEST POINT PREDICTION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F2-A — FREEZE FD002 CANONICAL TEST POINT PREDICTION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Safety barrier
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError(
        "Official TEST-label safety marker is missing."
    )

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError(
        "Official TEST labels have already been marked as read. Stop."
    )

# ---------------------------------------------------------------
# 2. Exact deterministic-lineage contract from C5-A4
# ---------------------------------------------------------------

required_prediction_names_628hc5f2a = [
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
    "TEST_DET_628FBR1",
    "test_prediction_628fb",
    "test_prediction_628hc0r2",
]

missing_628hc5f2a = [
    name
    for name in required_prediction_names_628hc5f2a
    if name not in globals()
]

if missing_628hc5f2a:
    raise RuntimeError(
        "Required deterministic prediction object(s) missing: "
        + ", ".join(missing_628hc5f2a)
    )

det_far2_628hc5f2a = np.asarray(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2,
    dtype=np.float64,
).reshape(-1)

det_fbr1_628hc5f2a = np.asarray(
    TEST_DET_628FBR1,
    dtype=np.float64,
).reshape(-1)

pred_fb_628hc5f2a = np.asarray(
    test_prediction_628fb,
    dtype=np.float64,
).reshape(-1)

pred_latest_628hc5f2a = np.asarray(
    test_prediction_628hc0r2,
    dtype=np.float64,
).reshape(-1)

all_shapes_259_628hc5f2a = bool(
    det_far2_628hc5f2a.shape == (259,)
    and det_fbr1_628hc5f2a.shape == (259,)
    and pred_fb_628hc5f2a.shape == (259,)
    and pred_latest_628hc5f2a.shape == (259,)
)

all_finite_628hc5f2a = bool(
    np.isfinite(det_far2_628hc5f2a).all()
    and np.isfinite(det_fbr1_628hc5f2a).all()
    and np.isfinite(pred_fb_628hc5f2a).all()
    and np.isfinite(pred_latest_628hc5f2a).all()
)

deterministic_cluster_exact_628hc5f2a = bool(
    np.array_equal(
        det_far2_628hc5f2a,
        det_fbr1_628hc5f2a,
    )
    and np.array_equal(
        det_far2_628hc5f2a,
        pred_fb_628hc5f2a,
    )
    and np.array_equal(
        det_far2_628hc5f2a,
        pred_latest_628hc5f2a,
    )
)

if not all_shapes_259_628hc5f2a:
    raise RuntimeError(
        "Deterministic prediction lineage does not have shape (259,)."
    )

if not all_finite_628hc5f2a:
    raise RuntimeError(
        "Deterministic prediction lineage contains non-finite values."
    )

if not deterministic_cluster_exact_628hc5f2a:
    raise RuntimeError(
        "Deterministic prediction lineage is not exactly identical. Stop."
    )

# ---------------------------------------------------------------
# 3. Canonical TEST point-prediction identity
# ---------------------------------------------------------------

FD002_CANONICAL_TEST_PREDICTION_628HC5F2A = (
    pred_latest_628hc5f2a.copy()
)

FD002_CANONICAL_TEST_PREDICTION_SOURCE_628HC5F2A = (
    "test_prediction_628hc0r2"
)

FD002_CANONICAL_TEST_PREDICTION_ROLE_628HC5F2A = (
    "deterministic endpoint RUL point prediction"
)

FD002_CANONICAL_TEST_PREDICTION_IS_MC_MEAN_628HC5F2A = False

# ---------------------------------------------------------------
# 4. SHA-256 helpers
# ---------------------------------------------------------------

def sha256_array_628hc5f2a(arr):

    a = np.ascontiguousarray(
        np.asarray(
            arr,
            dtype=np.float64,
        )
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def sha256_file_628hc5f2a(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


prediction_hash_628hc5f2a = (
    sha256_array_628hc5f2a(
        FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
    )
)

# ---------------------------------------------------------------
# 5. Freeze directory
# ---------------------------------------------------------------

freeze_dir_628hc5f2a = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_reliability_signals\fd002_c5"
)

freeze_dir_628hc5f2a.mkdir(
    parents=True,
    exist_ok=True,
)

npz_path_628hc5f2a = (
    freeze_dir_628hc5f2a
    / "fd002_c5_canonical_test_prediction.npz"
)

csv_path_628hc5f2a = (
    freeze_dir_628hc5f2a
    / "fd002_c5_canonical_test_prediction.csv"
)

metadata_path_628hc5f2a = (
    freeze_dir_628hc5f2a
    / "fd002_c5_canonical_test_prediction_metadata.json"
)

# ---------------------------------------------------------------
# 6. Endpoint identity
# ---------------------------------------------------------------

endpoint_position_1based_628hc5f2a = np.arange(
    1,
    260,
    dtype=np.int64,
)

prediction_table_628hc5f2a = pd.DataFrame(
    {
        "endpoint_position_1based":
            endpoint_position_1based_628hc5f2a,

        "prediction":
            FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,
    }
)

# ---------------------------------------------------------------
# 7. Write frozen prediction artifacts
# ---------------------------------------------------------------

np.savez_compressed(
    npz_path_628hc5f2a,

    prediction=
        FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,

    endpoint_position_1based=
        endpoint_position_1based_628hc5f2a,
)

prediction_table_628hc5f2a.to_csv(
    csv_path_628hc5f2a,
    index=False,
    float_format="%.17g",
)

npz_file_hash_628hc5f2a = (
    sha256_file_628hc5f2a(
        npz_path_628hc5f2a
    )
)

csv_file_hash_628hc5f2a = (
    sha256_file_628hc5f2a(
        csv_path_628hc5f2a
    )
)

# ---------------------------------------------------------------
# 8. Metadata contract
# ---------------------------------------------------------------

metadata_628hc5f2a = {

    "step":
        "6.28H-C5-F2-A",

    "dataset":
        "FD002",

    "n_test_endpoints":
        259,

    "prediction_semantic_role":
        FD002_CANONICAL_TEST_PREDICTION_ROLE_628HC5F2A,

    "canonical_source_object":
        FD002_CANONICAL_TEST_PREDICTION_SOURCE_628HC5F2A,

    "deterministic_lineage_objects": [
        "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
        "TEST_DET_628FBR1",
        "test_prediction_628fb",
        "test_prediction_628hc0r2",
    ],

    "deterministic_lineage_exact":
        True,

    "mc_mean_used_as_point_prediction":
        False,

    "prediction_shape":
        [259],

    "prediction_dtype":
        "float64",

    "prediction_sha256":
        prediction_hash_628hc5f2a,

    "npz_filename":
        npz_path_628hc5f2a.name,

    "csv_filename":
        csv_path_628hc5f2a.name,

    "npz_file_sha256":
        npz_file_hash_628hc5f2a,

    "csv_file_sha256":
        csv_file_hash_628hc5f2a,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "threshold_tuned_or_modified":
        False,

    "model_executed":
        False,

    "model_retrained":
        False,
}

metadata_path_628hc5f2a.write_text(
    json.dumps(
        metadata_628hc5f2a,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

metadata_file_hash_628hc5f2a = (
    sha256_file_628hc5f2a(
        metadata_path_628hc5f2a
    )
)

# ---------------------------------------------------------------
# 9. Immediate freeze audit
# ---------------------------------------------------------------

all_files_written_628hc5f2a = bool(
    npz_path_628hc5f2a.exists()
    and csv_path_628hc5f2a.exists()
    and metadata_path_628hc5f2a.exists()
)

if not all_files_written_628hc5f2a:
    raise RuntimeError(
        "C5-F2-A prediction freeze did not complete correctly."
    )

print("\n--- FROZEN TEST POINT-PREDICTION CONTRACT ---")

print(
    "Canonical prediction source :",
    FD002_CANONICAL_TEST_PREDICTION_SOURCE_628HC5F2A,
)

print(
    "Prediction semantic role    :",
    FD002_CANONICAL_TEST_PREDICTION_ROLE_628HC5F2A,
)

print(
    "Prediction shape            :",
    FD002_CANONICAL_TEST_PREDICTION_628HC5F2A.shape,
)

print(
    "Prediction min/median/mean/max:",
    float(
        np.min(
            FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
        )
    ),
    float(
        np.median(
            FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
        )
    ),
    float(
        np.mean(
            FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
        )
    ),
    float(
        np.max(
            FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
        )
    ),
)

print(
    "Deterministic lineage exact :",
    deterministic_cluster_exact_628hc5f2a,
)

print(
    "MC mean used as point prediction:",
    FD002_CANONICAL_TEST_PREDICTION_IS_MC_MEAN_628HC5F2A,
)

print("\n--- FREEZE DIRECTORY ---")
print(
    freeze_dir_628hc5f2a
)

print("\n--- WRITTEN ARTIFACTS ---")

print(
    npz_path_628hc5f2a.name,
    "| exists =",
    npz_path_628hc5f2a.exists(),
    "| bytes =",
    npz_path_628hc5f2a.stat().st_size,
)

print(
    csv_path_628hc5f2a.name,
    "| exists =",
    csv_path_628hc5f2a.exists(),
    "| bytes =",
    csv_path_628hc5f2a.stat().st_size,
)

print(
    metadata_path_628hc5f2a.name,
    "| exists =",
    metadata_path_628hc5f2a.exists(),
    "| bytes =",
    metadata_path_628hc5f2a.stat().st_size,
)

print("\n--- SHA-256 ---")

print(
    "prediction array :",
    prediction_hash_628hc5f2a,
)

print(
    "NPZ file         :",
    npz_file_hash_628hc5f2a,
)

print(
    "CSV file         :",
    csv_file_hash_628hc5f2a,
)

print(
    "metadata file    :",
    metadata_file_hash_628hc5f2a,
)

print("\n--- F2-A STATUS ---")

print(
    "All files written correctly :",
    all_files_written_628hc5f2a,
)

print("Official TEST labels read    : False")
print("TEST scored                  : False")
print("Prediction source frozen     : True")
print("Threshold tuned or modified  : False")
print("Model executed               : False")
print("Model retrained              : False")
print("Files written                : True")

print(
    "\nNext required step: "
    "C5-F2-B independent reload and exact verification."
)

print(
    "\n=== END STEP 6.28H-C5-F2-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-F2-B
# INDEPENDENT RELOAD + EXACT VERIFICATION OF CANONICAL TEST PREDICTION
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F2-B — INDEPENDENT RELOAD + EXACT PREDICTION VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Safety barrier
# ---------------------------------------------------------------

if "FD002_OFFICIAL_TEST_LABELS_READ_628A" not in globals():
    raise RuntimeError("Official TEST-label safety marker is missing.")

if bool(FD002_OFFICIAL_TEST_LABELS_READ_628A):
    raise RuntimeError("Official TEST labels are marked as read. Stop.")

# ---------------------------------------------------------------
# 2. Frozen artifact paths
# ---------------------------------------------------------------

freeze_dir_628hc5f2b = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_reliability_signals\fd002_c5"
)

npz_path_628hc5f2b = (
    freeze_dir_628hc5f2b
    / "fd002_c5_canonical_test_prediction.npz"
)

csv_path_628hc5f2b = (
    freeze_dir_628hc5f2b
    / "fd002_c5_canonical_test_prediction.csv"
)

metadata_path_628hc5f2b = (
    freeze_dir_628hc5f2b
    / "fd002_c5_canonical_test_prediction_metadata.json"
)

all_three_files_present_628hc5f2b = bool(
    npz_path_628hc5f2b.is_file()
    and csv_path_628hc5f2b.is_file()
    and metadata_path_628hc5f2b.is_file()
)

if not all_three_files_present_628hc5f2b:
    raise RuntimeError(
        "One or more C5-F2-A frozen prediction files are missing."
    )

# ---------------------------------------------------------------
# 3. Hash helpers
# ---------------------------------------------------------------

def sha256_array_628hc5f2b(arr):
    a = np.ascontiguousarray(
        np.asarray(arr, dtype=np.float64)
    )
    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def sha256_file_628hc5f2b(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()

# ---------------------------------------------------------------
# 4. Independent reload
# ---------------------------------------------------------------

with np.load(
    npz_path_628hc5f2b,
    allow_pickle=False
) as z:

    npz_keys_628hc5f2b = tuple(
        sorted(z.files)
    )

    prediction_reload_628hc5f2b = np.asarray(
        z["prediction"],
        dtype=np.float64,
    ).copy()

    endpoint_reload_628hc5f2b = np.asarray(
        z["endpoint_position_1based"],
        dtype=np.int64,
    ).copy()

csv_reload_628hc5f2b = pd.read_csv(
    csv_path_628hc5f2b
)

metadata_reload_628hc5f2b = json.loads(
    metadata_path_628hc5f2b.read_text(
        encoding="utf-8"
    )
)

# ---------------------------------------------------------------
# 5. Reloaded artifact contracts
# ---------------------------------------------------------------

expected_npz_keys_628hc5f2b = (
    "endpoint_position_1based",
    "prediction",
)

npz_keys_exact_628hc5f2b = bool(
    npz_keys_628hc5f2b
    == expected_npz_keys_628hc5f2b
)

prediction_shape_259_628hc5f2b = bool(
    prediction_reload_628hc5f2b.shape
    == (259,)
)

endpoint_shape_259_628hc5f2b = bool(
    endpoint_reload_628hc5f2b.shape
    == (259,)
)

prediction_finite_628hc5f2b = bool(
    np.isfinite(
        prediction_reload_628hc5f2b
    ).all()
)

endpoint_order_exact_628hc5f2b = bool(
    np.array_equal(
        endpoint_reload_628hc5f2b,
        np.arange(
            1,
            260,
            dtype=np.int64,
        ),
    )
)

# ---------------------------------------------------------------
# 6. File and array hashes
# ---------------------------------------------------------------

prediction_array_hash_reload_628hc5f2b = (
    sha256_array_628hc5f2b(
        prediction_reload_628hc5f2b
    )
)

npz_file_hash_reload_628hc5f2b = (
    sha256_file_628hc5f2b(
        npz_path_628hc5f2b
    )
)

csv_file_hash_reload_628hc5f2b = (
    sha256_file_628hc5f2b(
        csv_path_628hc5f2b
    )
)

metadata_file_hash_reload_628hc5f2b = (
    sha256_file_628hc5f2b(
        metadata_path_628hc5f2b
    )
)

prediction_hash_exact_vs_metadata_628hc5f2b = bool(
    prediction_array_hash_reload_628hc5f2b
    == metadata_reload_628hc5f2b[
        "prediction_sha256"
    ]
)

npz_hash_exact_vs_metadata_628hc5f2b = bool(
    npz_file_hash_reload_628hc5f2b
    == metadata_reload_628hc5f2b[
        "npz_file_sha256"
    ]
)

csv_hash_exact_vs_metadata_628hc5f2b = bool(
    csv_file_hash_reload_628hc5f2b
    == metadata_reload_628hc5f2b[
        "csv_file_sha256"
    ]
)

# ---------------------------------------------------------------
# 7. Exact comparison with frozen in-memory prediction
# ---------------------------------------------------------------

if "FD002_CANONICAL_TEST_PREDICTION_628HC5F2A" not in globals():
    raise RuntimeError(
        "C5-F2-A canonical in-memory prediction is missing."
    )

prediction_memory_628hc5f2b = np.asarray(
    FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,
    dtype=np.float64,
).reshape(-1)

prediction_exact_vs_memory_628hc5f2b = bool(
    np.array_equal(
        prediction_reload_628hc5f2b,
        prediction_memory_628hc5f2b,
    )
)

prediction_hash_exact_vs_memory_628hc5f2b = bool(
    prediction_array_hash_reload_628hc5f2b
    == sha256_array_628hc5f2b(
        prediction_memory_628hc5f2b
    )
)

# ---------------------------------------------------------------
# 8. Historical deterministic-lineage identity
# ---------------------------------------------------------------

historical_names_628hc5f2b = [
    "TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2",
    "TEST_DET_628FBR1",
    "test_prediction_628fb",
    "test_prediction_628hc0r2",
]

historical_present_628hc5f2b = bool(
    all(
        name in globals()
        for name in historical_names_628hc5f2b
    )
)

if not historical_present_628hc5f2b:
    raise RuntimeError(
        "Required historical deterministic lineage objects are missing."
    )

hist_1_628hc5f2b = np.asarray(
    TEST_DETERMINISTIC_PREDICTIONS_FD002_628FAR2,
    dtype=np.float64,
).reshape(-1)

hist_2_628hc5f2b = np.asarray(
    TEST_DET_628FBR1,
    dtype=np.float64,
).reshape(-1)

hist_3_628hc5f2b = np.asarray(
    test_prediction_628fb,
    dtype=np.float64,
).reshape(-1)

hist_4_628hc5f2b = np.asarray(
    test_prediction_628hc0r2,
    dtype=np.float64,
).reshape(-1)

historical_cluster_exact_628hc5f2b = bool(
    np.array_equal(
        prediction_reload_628hc5f2b,
        hist_1_628hc5f2b,
    )
    and np.array_equal(
        prediction_reload_628hc5f2b,
        hist_2_628hc5f2b,
    )
    and np.array_equal(
        prediction_reload_628hc5f2b,
        hist_3_628hc5f2b,
    )
    and np.array_equal(
        prediction_reload_628hc5f2b,
        hist_4_628hc5f2b,
    )
)

# ---------------------------------------------------------------
# 9. CSV verification
# ---------------------------------------------------------------

expected_csv_columns_628hc5f2b = [
    "endpoint_position_1based",
    "prediction",
]

csv_columns_exact_628hc5f2b = bool(
    list(
        csv_reload_628hc5f2b.columns
    )
    == expected_csv_columns_628hc5f2b
)

csv_rows_259_628hc5f2b = bool(
    len(csv_reload_628hc5f2b)
    == 259
)

csv_endpoint_order_exact_628hc5f2b = bool(
    np.array_equal(
        csv_reload_628hc5f2b[
            "endpoint_position_1based"
        ].to_numpy(
            dtype=np.int64
        ),
        endpoint_reload_628hc5f2b,
    )
)

csv_prediction_628hc5f2b = (
    csv_reload_628hc5f2b[
        "prediction"
    ].to_numpy(
        dtype=np.float64
    )
)

csv_prediction_exact_628hc5f2b = bool(
    np.array_equal(
        csv_prediction_628hc5f2b,
        prediction_reload_628hc5f2b,
    )
)

csv_prediction_close_1e15_628hc5f2b = bool(
    np.allclose(
        csv_prediction_628hc5f2b,
        prediction_reload_628hc5f2b,
        rtol=0.0,
        atol=1e-15,
    )
)

# ---------------------------------------------------------------
# 10. Metadata contract
# ---------------------------------------------------------------

metadata_contract_exact_628hc5f2b = bool(

    metadata_reload_628hc5f2b[
        "step"
    ] == "6.28H-C5-F2-A"

    and metadata_reload_628hc5f2b[
        "dataset"
    ] == "FD002"

    and metadata_reload_628hc5f2b[
        "n_test_endpoints"
    ] == 259

    and metadata_reload_628hc5f2b[
        "prediction_semantic_role"
    ] == "deterministic endpoint RUL point prediction"

    and metadata_reload_628hc5f2b[
        "canonical_source_object"
    ] == "test_prediction_628hc0r2"

    and metadata_reload_628hc5f2b[
        "deterministic_lineage_exact"
    ] is True

    and metadata_reload_628hc5f2b[
        "mc_mean_used_as_point_prediction"
    ] is False

    and metadata_reload_628hc5f2b[
        "prediction_shape"
    ] == [259]

    and metadata_reload_628hc5f2b[
        "prediction_dtype"
    ] == "float64"

    and metadata_reload_628hc5f2b[
        "official_test_labels_read"
    ] is False

    and metadata_reload_628hc5f2b[
        "test_scored"
    ] is False

    and metadata_reload_628hc5f2b[
        "threshold_tuned_or_modified"
    ] is False

    and metadata_reload_628hc5f2b[
        "model_executed"
    ] is False

    and metadata_reload_628hc5f2b[
        "model_retrained"
    ] is False
)

# ---------------------------------------------------------------
# 11. Same-session hash confirmation
# ---------------------------------------------------------------

same_session_npz_hash_exact_628hc5f2b = bool(
    "npz_file_hash_628hc5f2a" not in globals()
    or npz_file_hash_reload_628hc5f2b
       == npz_file_hash_628hc5f2a
)

same_session_csv_hash_exact_628hc5f2b = bool(
    "csv_file_hash_628hc5f2a" not in globals()
    or csv_file_hash_reload_628hc5f2b
       == csv_file_hash_628hc5f2a
)

same_session_metadata_hash_exact_628hc5f2b = bool(
    "metadata_file_hash_628hc5f2a" not in globals()
    or metadata_file_hash_reload_628hc5f2b
       == metadata_file_hash_628hc5f2a
)

# ---------------------------------------------------------------
# 12. Audit
# ---------------------------------------------------------------

audit_628hc5f2b = pd.DataFrame(
    [
        (
            "all_three_F2A_files_present",
            all_three_files_present_628hc5f2b,
        ),
        (
            "npz_keys_exact",
            npz_keys_exact_628hc5f2b,
        ),
        (
            "prediction_shape_259",
            prediction_shape_259_628hc5f2b,
        ),
        (
            "endpoint_shape_259",
            endpoint_shape_259_628hc5f2b,
        ),
        (
            "prediction_finite",
            prediction_finite_628hc5f2b,
        ),
        (
            "endpoint_order_exact",
            endpoint_order_exact_628hc5f2b,
        ),
        (
            "prediction_hash_exact_vs_metadata",
            prediction_hash_exact_vs_metadata_628hc5f2b,
        ),
        (
            "npz_hash_exact_vs_metadata",
            npz_hash_exact_vs_metadata_628hc5f2b,
        ),
        (
            "csv_hash_exact_vs_metadata",
            csv_hash_exact_vs_metadata_628hc5f2b,
        ),
        (
            "prediction_exact_vs_memory",
            prediction_exact_vs_memory_628hc5f2b,
        ),
        (
            "prediction_hash_exact_vs_memory",
            prediction_hash_exact_vs_memory_628hc5f2b,
        ),
        (
            "historical_deterministic_cluster_exact",
            historical_cluster_exact_628hc5f2b,
        ),
        (
            "csv_columns_exact",
            csv_columns_exact_628hc5f2b,
        ),
        (
            "csv_rows_259",
            csv_rows_259_628hc5f2b,
        ),
        (
            "csv_endpoint_order_exact",
            csv_endpoint_order_exact_628hc5f2b,
        ),
        (
            "csv_prediction_exact",
            csv_prediction_exact_628hc5f2b,
        ),
        (
            "csv_prediction_close_1e15",
            csv_prediction_close_1e15_628hc5f2b,
        ),
        (
            "metadata_contract_exact",
            metadata_contract_exact_628hc5f2b,
        ),
        (
            "same_session_npz_hash_exact",
            same_session_npz_hash_exact_628hc5f2b,
        ),
        (
            "same_session_csv_hash_exact",
            same_session_csv_hash_exact_628hc5f2b,
        ),
        (
            "same_session_metadata_hash_exact",
            same_session_metadata_hash_exact_628hc5f2b,
        ),
        (
            "official_test_label_marker_false",
            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
                is False
            ),
        ),
        (
            "no_test_file_read",
            True,
        ),
        (
            "no_test_label_read",
            True,
        ),
        (
            "no_test_scoring",
            True,
        ),
        (
            "no_threshold_tuning",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_files_modified",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C5-F2-B INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc5f2b
)

all_checks_passed_628hc5f2b = bool(
    audit_628hc5f2b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5f2b:
    raise RuntimeError(
        "C5-F2-B independent reload verification failed."
    )

# ---------------------------------------------------------------
# 13. Final summary
# ---------------------------------------------------------------

print("\n" + "=" * 94)
print("STEP 6.28H-C5-F2-B FINAL SUMMARY")
print("=" * 94)

print(
    "Canonical prediction source      :",
    metadata_reload_628hc5f2b[
        "canonical_source_object"
    ],
)

print(
    "Canonical prediction shape       :",
    prediction_reload_628hc5f2b.shape,
)

print(
    "Prediction min/median/mean/max   :",
    float(
        np.min(
            prediction_reload_628hc5f2b
        )
    ),
    float(
        np.median(
            prediction_reload_628hc5f2b
        )
    ),
    float(
        np.mean(
            prediction_reload_628hc5f2b
        )
    ),
    float(
        np.max(
            prediction_reload_628hc5f2b
        )
    ),
)

print(
    "Prediction array SHA-256         :",
    prediction_array_hash_reload_628hc5f2b,
)

print(
    "Historical deterministic lineage:",
    historical_cluster_exact_628hc5f2b,
)

print(
    "NPZ file hash exact              :",
    npz_hash_exact_vs_metadata_628hc5f2b,
)

print(
    "CSV file hash exact              :",
    csv_hash_exact_vs_metadata_628hc5f2b,
)

print(
    "Prediction exact vs memory       :",
    prediction_exact_vs_memory_628hc5f2b,
)

print(
    "Metadata contract exact          :",
    metadata_contract_exact_628hc5f2b,
)

print(
    "Independent reload verified      :",
    all_checks_passed_628hc5f2b,
)

print("Official TEST labels read        : False")
print("TEST scored                      : False")
print("Prediction source modified       : False")
print("Threshold tuned or modified      : False")
print("Model executed                   : False")
print("Model retrained                  : False")
print("Files modified                   : False")

print(
    "\n=== END STEP 6.28H-C5-F2-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-F2-B-R1
# CSV FLOAT64 ROUND-TRIP RECONCILIATION
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F2-B-R1 — CSV FLOAT64 ROUND-TRIP RECONCILIATION")
print("=" * 94)

# Canonical operational prediction remains the NPZ-reloaded float64 vector.
pred_npz_628hc5f2br1 = np.asarray(
    prediction_reload_628hc5f2b,
    dtype=np.float64,
).reshape(-1)

pred_csv_628hc5f2br1 = np.asarray(
    csv_prediction_628hc5f2b,
    dtype=np.float64,
).reshape(-1)

if pred_npz_628hc5f2br1.shape != (259,):
    raise RuntimeError("NPZ prediction shape is not (259,).")

if pred_csv_628hc5f2br1.shape != (259,):
    raise RuntimeError("CSV prediction shape is not (259,).")

abs_diff_628hc5f2br1 = np.abs(
    pred_csv_628hc5f2br1 - pred_npz_628hc5f2br1
)

max_abs_diff_628hc5f2br1 = float(np.max(abs_diff_628hc5f2br1))
mean_abs_diff_628hc5f2br1 = float(np.mean(abs_diff_628hc5f2br1))
median_abs_diff_628hc5f2br1 = float(np.median(abs_diff_628hc5f2br1))
q95_abs_diff_628hc5f2br1 = float(np.quantile(abs_diff_628hc5f2br1, 0.95))

nonzero_diff_count_628hc5f2br1 = int(
    np.count_nonzero(abs_diff_628hc5f2br1)
)

# ULP-aware diagnostic.
spacing_628hc5f2br1 = np.spacing(
    np.abs(pred_npz_628hc5f2br1)
)

spacing_floor_628hc5f2br1 = np.maximum(
    spacing_628hc5f2br1,
    np.finfo(np.float64).tiny,
)

ulp_error_628hc5f2br1 = (
    abs_diff_628hc5f2br1 / spacing_floor_628hc5f2br1
)

max_ulp_error_628hc5f2br1 = float(
    np.max(ulp_error_628hc5f2br1)
)

# CSV is a decimal interchange artifact, not the canonical binary source.
# This remains a very strict bound relative to predictions of magnitude ~3–126.
CSV_ROUNDTRIP_ATOL_628HC5F2BR1 = 1e-12

csv_roundtrip_semantic_exact_628hc5f2br1 = bool(
    np.allclose(
        pred_csv_628hc5f2br1,
        pred_npz_628hc5f2br1,
        rtol=0.0,
        atol=CSV_ROUNDTRIP_ATOL_628HC5F2BR1,
    )
)

print("\n--- CSV NUMERICAL ROUND-TRIP DIAGNOSTIC ---")
print("Different parsed elements         :", nonzero_diff_count_628hc5f2br1)
print("Max absolute difference           :", max_abs_diff_628hc5f2br1)
print("Mean absolute difference          :", mean_abs_diff_628hc5f2br1)
print("Median absolute difference        :", median_abs_diff_628hc5f2br1)
print("q95 absolute difference           :", q95_abs_diff_628hc5f2br1)
print("Max ULP error                     :", max_ulp_error_628hc5f2br1)
print("Required CSV semantic atol        :", CSV_ROUNDTRIP_ATOL_628HC5F2BR1)
print("CSV semantic round-trip accepted :", csv_roundtrip_semantic_exact_628hc5f2br1)

audit_628hc5f2br1 = pd.DataFrame(
    [
        (
            "npz_prediction_exact_vs_memory",
            bool(prediction_exact_vs_memory_628hc5f2b),
        ),
        (
            "npz_prediction_hash_exact_vs_memory",
            bool(prediction_hash_exact_vs_memory_628hc5f2b),
        ),
        (
            "historical_deterministic_cluster_exact",
            bool(historical_cluster_exact_628hc5f2b),
        ),
        (
            "npz_file_hash_exact",
            bool(npz_hash_exact_vs_metadata_628hc5f2b),
        ),
        (
            "csv_file_hash_exact",
            bool(csv_hash_exact_vs_metadata_628hc5f2b),
        ),
        (
            "csv_endpoint_order_exact",
            bool(csv_endpoint_order_exact_628hc5f2b),
        ),
        (
            "csv_prediction_semantic_roundtrip_1e12",
            csv_roundtrip_semantic_exact_628hc5f2br1,
        ),
        (
            "metadata_contract_exact",
            bool(metadata_contract_exact_628hc5f2b),
        ),
        (
            "official_test_label_marker_false",
            bool(FD002_OFFICIAL_TEST_LABELS_READ_628A is False),
        ),
        ("no_test_file_read", True),
        ("no_test_label_read", True),
        ("no_test_scoring", True),
        ("no_threshold_tuning", True),
        ("no_threshold_modification", True),
        ("no_model_execution", True),
        ("no_model_retraining", True),
        ("no_files_modified", True),
    ],
    columns=["check", "passed"],
)

print("\n--- C5-F2-B-R1 AUDIT ---")
display(audit_628hc5f2br1)

all_checks_passed_628hc5f2br1 = bool(
    audit_628hc5f2br1["passed"].all()
)

if not all_checks_passed_628hc5f2br1:
    raise RuntimeError(
        "C5-F2-B-R1 failed: CSV discrepancy exceeds the "
        "strict semantic round-trip contract."
    )

print("\n" + "=" * 94)
print("STEP 6.28H-C5-F2-B-R1 FINAL SUMMARY")
print("=" * 94)

print("Canonical binary prediction exact : True")
print("Canonical prediction hash exact   : True")
print("Historical lineage exact          : True")
print("CSV file hash exact               : True")
print(
    "CSV numerical round-trip accepted :",
    csv_roundtrip_semantic_exact_628hc5f2br1,
)
print(
    "CSV max absolute difference        :",
    max_abs_diff_628hc5f2br1,
)
print(
    "Independent reload verified        :",
    all_checks_passed_628hc5f2br1,
)

print("Official TEST labels read          : False")
print("TEST scored                        : False")
print("Threshold tuned or modified        : False")
print("Model executed                     : False")
print("Model retrained                    : False")
print("Files modified                     : False")

print("\n=== END STEP 6.28H-C5-F2-B-R1 ===")

In [ ]:
# ================================================================
# STEP 6.28H-C5-A5
# FINAL PRE-SCORING FD002 TEST CONTRACT LOCK
# ================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-A5 — FINAL PRE-SCORING FD002 TEST CONTRACT LOCK")
print("=" * 94)

# ----------------------------------------------------------------
# 1. Frozen constants / canonical in-memory objects only
# ----------------------------------------------------------------
EXPECTED_N_628HC5A5 = 259
FROZEN_THRESHOLD_628HC5A5 = 0.8003345892330226

EXPECTED_RISK_SHA256_628HC5A5 = (
    "fbd40b69ab28807738d64b374481ac4e71482cd6bede0909087af5aa729ed3ae"
)

EXPECTED_PRED_SHA256_628HC5A5 = (
    "10470072f7f027e86d5ec5dfb4a2c8077c13ad914e042e4a7721f066edf37d0b"
)

EXPECTED_ACCEPT_SHA256_628HC5A5 = (
    "8aa52655b56ab24c09941eb0b545b8a7dceabaea10619b7999b247ff885520be"
)

EXPECTED_ABSTAIN_SHA256_628HC5A5 = (
    "ecf4e3738b4732d25477e2d67a5517859bda83a602ece8983fd5526388b17559"
)

risk_628hc5a5 = np.asarray(
    FD002_TEST_SELECTIVE_RISK_628FCR2,
    dtype=np.float64,
).reshape(-1)

prediction_628hc5a5 = np.asarray(
    pred_npz_628hc5f2br1,
    dtype=np.float64,
).reshape(-1)

endpoint_628hc5a5 = np.arange(
    1,
    EXPECTED_N_628HC5A5 + 1,
    dtype=np.int64,
)

accept_628hc5a5 = (
    risk_628hc5a5 <= FROZEN_THRESHOLD_628HC5A5
)

abstain_628hc5a5 = ~accept_628hc5a5

# ----------------------------------------------------------------
# 2. Hash helper
# ----------------------------------------------------------------
def sha256_array_628hc5a5(arr):
    a = np.ascontiguousarray(arr)
    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


risk_sha_628hc5a5 = sha256_array_628hc5a5(
    risk_628hc5a5
)

prediction_sha_628hc5a5 = sha256_array_628hc5a5(
    prediction_628hc5a5
)

accept_sha_628hc5a5 = sha256_array_628hc5a5(
    accept_628hc5a5
)

abstain_sha_628hc5a5 = sha256_array_628hc5a5(
    abstain_628hc5a5
)

# ----------------------------------------------------------------
# 3. Final aligned label-free TEST contract
# ----------------------------------------------------------------
FD002_TEST_PRESCORING_TABLE_628HC5A5 = pd.DataFrame(
    {
        "endpoint_position_1based": endpoint_628hc5a5,
        "point_prediction": prediction_628hc5a5,
        "reliability_risk": risk_628hc5a5,
        "frozen_risk_threshold": np.full(
            EXPECTED_N_628HC5A5,
            FROZEN_THRESHOLD_628HC5A5,
            dtype=np.float64,
        ),
        "accepted": accept_628hc5a5,
        "abstained": abstain_628hc5a5,
    }
)

accepted_count_628hc5a5 = int(
    np.sum(accept_628hc5a5)
)

abstained_count_628hc5a5 = int(
    np.sum(abstain_628hc5a5)
)

realized_coverage_628hc5a5 = (
    accepted_count_628hc5a5
    / EXPECTED_N_628HC5A5
)

threshold_ties_628hc5a5 = int(
    np.sum(
        risk_628hc5a5
        == FROZEN_THRESHOLD_628HC5A5
    )
)

# ----------------------------------------------------------------
# 4. Combined pre-scoring fingerprint
# ----------------------------------------------------------------
contract_hasher_628hc5a5 = hashlib.sha256()

contract_hasher_628hc5a5.update(
    np.ascontiguousarray(
        endpoint_628hc5a5
    ).tobytes(order="C")
)

contract_hasher_628hc5a5.update(
    np.ascontiguousarray(
        prediction_628hc5a5
    ).tobytes(order="C")
)

contract_hasher_628hc5a5.update(
    np.ascontiguousarray(
        risk_628hc5a5
    ).tobytes(order="C")
)

contract_hasher_628hc5a5.update(
    np.ascontiguousarray(
        accept_628hc5a5
    ).tobytes(order="C")
)

contract_hasher_628hc5a5.update(
    np.ascontiguousarray(
        abstain_628hc5a5
    ).tobytes(order="C")
)

contract_hasher_628hc5a5.update(
    np.asarray(
        [FROZEN_THRESHOLD_628HC5A5],
        dtype=np.float64,
    ).tobytes(order="C")
)

FD002_TEST_PRESCORING_SHA256_628HC5A5 = (
    contract_hasher_628hc5a5.hexdigest()
)

FD002_TEST_PRESCORING_CONTRACT_628HC5A5 = {
    "reliability_architecture":
        "uncertainty_only",

    "operating_policy":
        "uncertainty_only_fixed_threshold",

    "canonical_risk_source":
        "FD002_TEST_SELECTIVE_RISK_628FCR2",

    "canonical_prediction_source":
        "frozen C5-F2 deterministic endpoint prediction",

    "n_test_endpoints":
        EXPECTED_N_628HC5A5,

    "frozen_risk_threshold":
        FROZEN_THRESHOLD_628HC5A5,

    "accepted_test_endpoints":
        accepted_count_628hc5a5,

    "abstained_test_endpoints":
        abstained_count_628hc5a5,

    "realized_test_coverage":
        realized_coverage_628hc5a5,

    "risk_sha256":
        risk_sha_628hc5a5,

    "prediction_sha256":
        prediction_sha_628hc5a5,

    "accept_mask_sha256":
        accept_sha_628hc5a5,

    "abstain_mask_sha256":
        abstain_sha_628hc5a5,

    "prescoring_contract_sha256":
        FD002_TEST_PRESCORING_SHA256_628HC5A5,
}

# ----------------------------------------------------------------
# 5. Audit
# ----------------------------------------------------------------
audit_628hc5a5 = pd.DataFrame(
    [
        (
            "risk_shape_259",
            risk_628hc5a5.shape == (259,),
        ),
        (
            "prediction_shape_259",
            prediction_628hc5a5.shape == (259,),
        ),
        (
            "endpoint_order_1_to_259",
            np.array_equal(
                endpoint_628hc5a5,
                np.arange(
                    1,
                    260,
                    dtype=np.int64,
                ),
            ),
        ),
        (
            "risk_finite",
            np.isfinite(
                risk_628hc5a5
            ).all(),
        ),
        (
            "prediction_finite",
            np.isfinite(
                prediction_628hc5a5
            ).all(),
        ),
        (
            "risk_within_0_1",
            bool(
                np.all(
                    (risk_628hc5a5 >= 0.0)
                    & (risk_628hc5a5 <= 1.0)
                )
            ),
        ),
        (
            "risk_sha256_exact",
            risk_sha_628hc5a5
            == EXPECTED_RISK_SHA256_628HC5A5,
        ),
        (
            "prediction_sha256_exact",
            prediction_sha_628hc5a5
            == EXPECTED_PRED_SHA256_628HC5A5,
        ),
        (
            "accept_sha256_exact",
            accept_sha_628hc5a5
            == EXPECTED_ACCEPT_SHA256_628HC5A5,
        ),
        (
            "abstain_sha256_exact",
            abstain_sha_628hc5a5
            == EXPECTED_ABSTAIN_SHA256_628HC5A5,
        ),
        (
            "accept_abstain_complements",
            np.array_equal(
                abstain_628hc5a5,
                ~accept_628hc5a5,
            ),
        ),
        (
            "accepted_count_211",
            accepted_count_628hc5a5 == 211,
        ),
        (
            "abstained_count_48",
            abstained_count_628hc5a5 == 48,
        ),
        (
            "decision_count_259",
            (
                accepted_count_628hc5a5
                + abstained_count_628hc5a5
            )
            == 259,
        ),
        (
            "threshold_ties_zero",
            threshold_ties_628hc5a5 == 0,
        ),
        (
            "frozen_threshold_exact",
            FROZEN_THRESHOLD_628HC5A5
            == 0.8003345892330226,
        ),
        (
            "f2_prediction_reload_verified",
            bool(
                all_checks_passed_628hc5f2br1
            ),
        ),
        (
            "official_test_label_marker_false",
            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
                is False
            ),
        ),
        (
            "no_test_file_read",
            True,
        ),
        (
            "no_test_label_read",
            True,
        ),
        (
            "no_test_scoring",
            True,
        ),
        (
            "no_threshold_tuning",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C5-A5 FINAL PRE-SCORING AUDIT ---"
)

display(
    audit_628hc5a5
)

all_checks_passed_628hc5a5 = bool(
    audit_628hc5a5[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5a5:
    raise RuntimeError(
        "C5-A5 final pre-scoring contract "
        "lock failed. Do not read TEST labels."
    )

# ----------------------------------------------------------------
# 6. Final summary
# ----------------------------------------------------------------
print("\n" + "=" * 94)
print(
    "STEP 6.28H-C5-A5 FINAL SUMMARY"
)
print("=" * 94)

print(
    "Reliability architecture       : "
    "uncertainty_only"
)

print(
    "Operating policy               : "
    "uncertainty_only_fixed_threshold"
)

print(
    "Canonical TEST risk source     : "
    "FD002_TEST_SELECTIVE_RISK_628FCR2"
)

print(
    "Canonical TEST prediction      : "
    "frozen deterministic endpoint prediction"
)

print(
    "TEST endpoints                 :",
    EXPECTED_N_628HC5A5,
)

print(
    "Frozen exact risk threshold    :",
    FROZEN_THRESHOLD_628HC5A5,
)

print(
    "Accepted TEST endpoints        :",
    accepted_count_628hc5a5,
)

print(
    "Abstained TEST endpoints       :",
    abstained_count_628hc5a5,
)

print(
    "Realized TEST coverage         :",
    realized_coverage_628hc5a5,
)

print(
    "Threshold ties                 :",
    threshold_ties_628hc5a5,
)

print(
    "Risk SHA-256 exact             : True"
)

print(
    "Prediction SHA-256 exact       : True"
)

print(
    "Decision-mask SHA-256 exact    : True"
)

print(
    "Pre-scoring contract SHA-256   :",
    FD002_TEST_PRESCORING_SHA256_628HC5A5,
)

print(
    "All checks passed              :",
    all_checks_passed_628hc5a5,
)

print(
    "Official TEST labels read      : False"
)

print(
    "TEST scored                    : False"
)

print(
    "Threshold tuned or modified    : False"
)

print(
    "Model executed                 : False"
)

print(
    "Model retrained                : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.28H-C5-A5 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-S1
# ONE-TIME OFFICIAL FD002 TEST SCORING
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-S1 — ONE-TIME OFFICIAL FD002 TEST SCORING")
print("=" * 94)

N_628HC5S1 = 259
RUL_CAP_628HC5S1 = 125.0
THRESHOLD_628HC5S1 = 0.8003345892330226

PRESCORING_SHA_628HC5S1 = (
    "291b1bdfbea2f753133868573ba27916653e1de4d4bf1906b4ae86689fe3ec16"
)

# ---------------------------------------------------------------
# 1. Fail closed BEFORE reading official TEST labels
# ---------------------------------------------------------------
precheck_628hc5s1 = pd.DataFrame(
    [
        (
            "A5_all_checks_passed",
            bool(all_checks_passed_628hc5a5),
        ),
        (
            "prescoring_contract_hash_exact",
            FD002_TEST_PRESCORING_SHA256_628HC5A5
            == PRESCORING_SHA_628HC5S1,
        ),
        (
            "prediction_shape_259",
            prediction_628hc5a5.shape == (259,),
        ),
        (
            "risk_shape_259",
            risk_628hc5a5.shape == (259,),
        ),
        (
            "accept_shape_259",
            accept_628hc5a5.shape == (259,),
        ),
        (
            "abstain_shape_259",
            abstain_628hc5a5.shape == (259,),
        ),
        (
            "accepted_count_211",
            int(np.sum(accept_628hc5a5)) == 211,
        ),
        (
            "abstained_count_48",
            int(np.sum(abstain_628hc5a5)) == 48,
        ),
        (
            "frozen_threshold_exact",
            THRESHOLD_628HC5S1 == 0.8003345892330226,
        ),
    ],
    columns=["check", "passed"],
)

print("\n--- PRE-READ CONTRACT ---")
display(precheck_628hc5s1)

if not bool(precheck_628hc5s1["passed"].all()):
    raise RuntimeError(
        "C5-S1 pre-read contract failed. "
        "Official TEST labels were NOT read."
    )

# ---------------------------------------------------------------
# 2. ONE-TIME official RUL_FD002 read
# ---------------------------------------------------------------
rul_path_628hc5s1 = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\RUL_FD002.txt"
)

if not rul_path_628hc5s1.is_file():
    raise FileNotFoundError(
        f"Official FD002 RUL file not found: "
        f"{rul_path_628hc5s1}"
    )

y_raw_628hc5s1 = np.loadtxt(
    rul_path_628hc5s1,
    dtype=np.float64,
).reshape(-1)

# The official TEST-label file has now been read.
FD002_OFFICIAL_TEST_LABELS_READ_628A = True

if y_raw_628hc5s1.shape != (N_628HC5S1,):
    raise RuntimeError(
        "Official FD002 TEST-label length is not 259."
    )

if not np.isfinite(y_raw_628hc5s1).all():
    raise RuntimeError(
        "Official FD002 TEST labels contain non-finite values."
    )

if np.any(y_raw_628hc5s1 < 0.0):
    raise RuntimeError(
        "Official FD002 TEST labels contain negative RUL values."
    )

# Same piecewise-capped target semantics used in development.
y_eval_628hc5s1 = np.minimum(
    y_raw_628hc5s1,
    RUL_CAP_628HC5S1,
)

pred_628hc5s1 = np.asarray(
    prediction_628hc5a5,
    dtype=np.float64,
).reshape(-1)

risk_628hc5s1 = np.asarray(
    risk_628hc5a5,
    dtype=np.float64,
).reshape(-1)

accept_628hc5s1 = np.asarray(
    accept_628hc5a5,
    dtype=bool,
).reshape(-1)

abstain_628hc5s1 = np.asarray(
    abstain_628hc5a5,
    dtype=bool,
).reshape(-1)

# ---------------------------------------------------------------
# 3. Metric helper
# ---------------------------------------------------------------
def metric_block_628hc5s1(mask):
    yt = y_eval_628hc5s1[mask]
    yp = pred_628hc5s1[mask]

    err = yp - yt

    nasa = np.where(
        err < 0.0,
        np.exp(-err / 13.0) - 1.0,
        np.exp(err / 10.0) - 1.0,
    )

    return {
        "n": int(mask.sum()),
        "mae": float(
            np.mean(
                np.abs(err)
            )
        ),
        "rmse": float(
            np.sqrt(
                np.mean(
                    err ** 2
                )
            )
        ),
        "nasa_score_total": float(
            np.sum(nasa)
        ),
        "nasa_score_mean": float(
            np.mean(nasa)
        ),
    }

all_mask_628hc5s1 = np.ones(
    N_628HC5S1,
    dtype=bool,
)

m_all_628hc5s1 = metric_block_628hc5s1(
    all_mask_628hc5s1
)

m_acc_628hc5s1 = metric_block_628hc5s1(
    accept_628hc5s1
)

m_abs_628hc5s1 = metric_block_628hc5s1(
    abstain_628hc5s1
)

# ---------------------------------------------------------------
# 4. Error / selective diagnostics
# ---------------------------------------------------------------
signed_error_628hc5s1 = (
    pred_628hc5s1
    - y_eval_628hc5s1
)

abs_error_628hc5s1 = np.abs(
    signed_error_628hc5s1
)

risk_error_spearman_628hc5s1 = float(
    pd.Series(
        risk_628hc5s1
    ).corr(
        pd.Series(
            abs_error_628hc5s1
        ),
        method="spearman",
    )
)

mae_reduction_628hc5s1 = (
    m_all_628hc5s1["mae"]
    - m_acc_628hc5s1["mae"]
)

rmse_reduction_628hc5s1 = (
    m_all_628hc5s1["rmse"]
    - m_acc_628hc5s1["rmse"]
)

relative_mae_reduction_628hc5s1 = (
    mae_reduction_628hc5s1
    / m_all_628hc5s1["mae"]
)

# ---------------------------------------------------------------
# 5. Final in-memory scoring artifacts
# ---------------------------------------------------------------
FD002_TEST_SCORING_TABLE_628HC5S1 = pd.DataFrame(
    {
        "endpoint_position_1based": np.arange(
            1,
            260,
            dtype=np.int64,
        ),
        "official_rul_raw":
            y_raw_628hc5s1,
        "evaluation_rul_capped_125":
            y_eval_628hc5s1,
        "point_prediction":
            pred_628hc5s1,
        "reliability_risk":
            risk_628hc5s1,
        "frozen_risk_threshold":
            THRESHOLD_628HC5S1,
        "accepted":
            accept_628hc5s1,
        "abstained":
            abstain_628hc5s1,
        "signed_error_pred_minus_true":
            signed_error_628hc5s1,
        "absolute_error":
            abs_error_628hc5s1,
    }
)

FD002_TEST_SCORE_SUMMARY_628HC5S1 = pd.DataFrame(
    [
        {
            "population": "all",
            **m_all_628hc5s1,
        },
        {
            "population": "accepted",
            **m_acc_628hc5s1,
        },
        {
            "population": "abstained",
            **m_abs_628hc5s1,
        },
    ]
)

# ---------------------------------------------------------------
# 6. Post-read audit
# ---------------------------------------------------------------
audit_628hc5s1 = pd.DataFrame(
    [
        (
            "official_label_marker_true",
            FD002_OFFICIAL_TEST_LABELS_READ_628A
            is True,
        ),
        (
            "official_labels_shape_259",
            y_raw_628hc5s1.shape == (259,),
        ),
        (
            "official_labels_finite",
            np.isfinite(
                y_raw_628hc5s1
            ).all(),
        ),
        (
            "evaluation_target_shape_259",
            y_eval_628hc5s1.shape == (259,),
        ),
        (
            "prediction_shape_259",
            pred_628hc5s1.shape == (259,),
        ),
        (
            "risk_shape_259",
            risk_628hc5s1.shape == (259,),
        ),
        (
            "accept_count_still_211",
            int(
                accept_628hc5s1.sum()
            )
            == 211,
        ),
        (
            "abstain_count_still_48",
            int(
                abstain_628hc5s1.sum()
            )
            == 48,
        ),
        (
            "threshold_unchanged",
            THRESHOLD_628HC5S1
            == 0.8003345892330226,
        ),
        (
            "prescoring_contract_unchanged",
            FD002_TEST_PRESCORING_SHA256_628HC5A5
            == PRESCORING_SHA_628HC5S1,
        ),
        (
            "no_threshold_tuning",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

# ---------------------------------------------------------------
# 7. Results
# ---------------------------------------------------------------
print("\n--- OFFICIAL TEST TARGET SUMMARY ---")

print(
    "Raw RUL min/median/mean/max :",
    float(
        np.min(
            y_raw_628hc5s1
        )
    ),
    float(
        np.median(
            y_raw_628hc5s1
        )
    ),
    float(
        np.mean(
            y_raw_628hc5s1
        )
    ),
    float(
        np.max(
            y_raw_628hc5s1
        )
    ),
)

print(
    "Frozen RUL cap              :",
    RUL_CAP_628HC5S1,
)

print(
    "Raw labels above cap        :",
    int(
        np.sum(
            y_raw_628hc5s1
            > RUL_CAP_628HC5S1
        )
    ),
)

print(
    "\n--- FINAL FD002 TEST PERFORMANCE ---"
)

display(
    FD002_TEST_SCORE_SUMMARY_628HC5S1
)

print(
    "\n--- SELECTIVE EFFECT ---"
)

print(
    "Realized coverage          :",
    float(
        np.mean(
            accept_628hc5s1
        )
    ),
)

print(
    "Full TEST MAE              :",
    m_all_628hc5s1["mae"],
)

print(
    "Accepted TEST MAE          :",
    m_acc_628hc5s1["mae"],
)

print(
    "Abstained TEST MAE         :",
    m_abs_628hc5s1["mae"],
)

print(
    "MAE reduction              :",
    mae_reduction_628hc5s1,
)

print(
    "Relative MAE reduction     :",
    relative_mae_reduction_628hc5s1,
)

print(
    "Full TEST RMSE             :",
    m_all_628hc5s1["rmse"],
)

print(
    "Accepted TEST RMSE         :",
    m_acc_628hc5s1["rmse"],
)

print(
    "Abstained TEST RMSE        :",
    m_abs_628hc5s1["rmse"],
)

print(
    "RMSE reduction             :",
    rmse_reduction_628hc5s1,
)

print(
    "Risk vs |error| Spearman   :",
    risk_error_spearman_628hc5s1,
)

print(
    "\n--- C5-S1 AUDIT ---"
)

display(
    audit_628hc5s1
)

all_checks_passed_628hc5s1 = bool(
    audit_628hc5s1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5s1:
    raise RuntimeError(
        "C5-S1 scoring audit failed. "
        "Do not reinterpret or retune "
        "the frozen policy."
    )

# ---------------------------------------------------------------
# 8. Final summary
# ---------------------------------------------------------------
print("\n" + "=" * 94)
print(
    "STEP 6.28H-C5-S1 FINAL SUMMARY"
)
print("=" * 94)

print(
    "Official TEST labels read           : True"
)

print(
    "Evaluation target                   : "
    "piecewise-capped RUL, cap=125"
)

print(
    "TEST endpoints                      :",
    N_628HC5S1,
)

print(
    "Frozen threshold                    :",
    THRESHOLD_628HC5S1,
)

print(
    "Accepted / abstained                :",
    f"{accept_628hc5s1.sum()} / "
    f"{abstain_628hc5s1.sum()}",
)

print(
    "Realized TEST coverage              :",
    float(
        np.mean(
            accept_628hc5s1
        )
    ),
)

print(
    "Full TEST MAE                       :",
    m_all_628hc5s1["mae"],
)

print(
    "Accepted TEST MAE                   :",
    m_acc_628hc5s1["mae"],
)

print(
    "Abstained TEST MAE                  :",
    m_abs_628hc5s1["mae"],
)

print(
    "MAE reduction                       :",
    mae_reduction_628hc5s1,
)

print(
    "Risk vs |error| Spearman            :",
    risk_error_spearman_628hc5s1,
)

print(
    "Threshold tuned or modified         : False"
)

print(
    "Model executed                      : False"
)

print(
    "Model retrained                     : False"
)

print(
    "Files written                       : False"
)

print(
    "\n=== END STEP 6.28H-C5-S1 ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-F3-A
# FREEZE OFFICIAL FD002 TEST SCORING RESULTS
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F3-A — FREEZE OFFICIAL FD002 TEST SCORING RESULTS")
print("=" * 94)

FREEZE_DIR_628HC5F3A = Path(
    r"C:\RUL_Reliability_Research\06_results\step6_reliability_signals\fd002_c5"
)

NPZ_PATH_628HC5F3A = (
    FREEZE_DIR_628HC5F3A
    / "fd002_c5_official_test_scoring.npz"
)

TABLE_CSV_PATH_628HC5F3A = (
    FREEZE_DIR_628HC5F3A
    / "fd002_c5_official_test_scoring.csv"
)

SUMMARY_CSV_PATH_628HC5F3A = (
    FREEZE_DIR_628HC5F3A
    / "fd002_c5_official_test_score_summary.csv"
)

META_PATH_628HC5F3A = (
    FREEZE_DIR_628HC5F3A
    / "fd002_c5_official_test_scoring_metadata.json"
)

# ---------------------------------------------------------------
# 1. Fail closed before writing anything
# ---------------------------------------------------------------
prewrite_628hc5f3a = pd.DataFrame(
    [
        (
            "S1_all_checks_passed",
            bool(all_checks_passed_628hc5s1),
        ),
        (
            "official_test_labels_read",
            FD002_OFFICIAL_TEST_LABELS_READ_628A is True,
        ),
        (
            "scoring_table_rows_259",
            len(FD002_TEST_SCORING_TABLE_628HC5S1) == 259,
        ),
        (
            "score_summary_rows_3",
            len(FD002_TEST_SCORE_SUMMARY_628HC5S1) == 3,
        ),
        (
            "prediction_shape_259",
            np.asarray(pred_628hc5s1).shape == (259,),
        ),
        (
            "risk_shape_259",
            np.asarray(risk_628hc5s1).shape == (259,),
        ),
        (
            "labels_raw_shape_259",
            np.asarray(y_raw_628hc5s1).shape == (259,),
        ),
        (
            "labels_eval_shape_259",
            np.asarray(y_eval_628hc5s1).shape == (259,),
        ),
        (
            "accepted_count_211",
            int(np.sum(accept_628hc5s1)) == 211,
        ),
        (
            "abstained_count_48",
            int(np.sum(abstain_628hc5s1)) == 48,
        ),
        (
            "threshold_exact",
            float(THRESHOLD_628HC5S1)
            == 0.8003345892330226,
        ),
        (
            "prescoring_contract_exact",
            FD002_TEST_PRESCORING_SHA256_628HC5A5
            == PRESCORING_SHA_628HC5S1,
        ),
    ],
    columns=["check", "passed"],
)

print("\n--- PRE-WRITE CONTRACT ---")
display(prewrite_628hc5f3a)

if not bool(prewrite_628hc5f3a["passed"].all()):
    raise RuntimeError(
        "C5-F3-A pre-write contract failed. "
        "No scoring artifacts were written."
    )

# ---------------------------------------------------------------
# 2. Canonicalize arrays
# ---------------------------------------------------------------
endpoint_628hc5f3a = np.arange(
    1,
    260,
    dtype=np.int64,
)

raw_rul_628hc5f3a = np.ascontiguousarray(
    y_raw_628hc5s1,
    dtype=np.float64,
)

eval_rul_628hc5f3a = np.ascontiguousarray(
    y_eval_628hc5s1,
    dtype=np.float64,
)

prediction_628hc5f3a = np.ascontiguousarray(
    pred_628hc5s1,
    dtype=np.float64,
)

risk_628hc5f3a = np.ascontiguousarray(
    risk_628hc5s1,
    dtype=np.float64,
)

accept_628hc5f3a = np.ascontiguousarray(
    accept_628hc5s1,
    dtype=np.bool_,
)

abstain_628hc5f3a = np.ascontiguousarray(
    abstain_628hc5s1,
    dtype=np.bool_,
)

signed_error_628hc5f3a = np.ascontiguousarray(
    signed_error_628hc5s1,
    dtype=np.float64,
)

absolute_error_628hc5f3a = np.ascontiguousarray(
    abs_error_628hc5s1,
    dtype=np.float64,
)

def sha256_array_628hc5f3a(arr):
    a = np.ascontiguousarray(arr)
    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()

array_hashes_628hc5f3a = {
    "endpoint_position_1based":
        sha256_array_628hc5f3a(endpoint_628hc5f3a),
    "official_rul_raw":
        sha256_array_628hc5f3a(raw_rul_628hc5f3a),
    "evaluation_rul_capped_125":
        sha256_array_628hc5f3a(eval_rul_628hc5f3a),
    "point_prediction":
        sha256_array_628hc5f3a(prediction_628hc5f3a),
    "reliability_risk":
        sha256_array_628hc5f3a(risk_628hc5f3a),
    "accept_mask":
        sha256_array_628hc5f3a(accept_628hc5f3a),
    "abstain_mask":
        sha256_array_628hc5f3a(abstain_628hc5f3a),
    "signed_error":
        sha256_array_628hc5f3a(signed_error_628hc5f3a),
    "absolute_error":
        sha256_array_628hc5f3a(absolute_error_628hc5f3a),
}

# ---------------------------------------------------------------
# 3. Freeze authoritative binary artifact
# ---------------------------------------------------------------
FREEZE_DIR_628HC5F3A.mkdir(
    parents=True,
    exist_ok=True,
)

np.savez_compressed(
    NPZ_PATH_628HC5F3A,
    endpoint_position_1based=endpoint_628hc5f3a,
    official_rul_raw=raw_rul_628hc5f3a,
    evaluation_rul_capped_125=eval_rul_628hc5f3a,
    point_prediction=prediction_628hc5f3a,
    reliability_risk=risk_628hc5f3a,
    accept_mask=accept_628hc5f3a,
    abstain_mask=abstain_628hc5f3a,
    signed_error=signed_error_628hc5f3a,
    absolute_error=absolute_error_628hc5f3a,
    frozen_risk_threshold=np.array(
        THRESHOLD_628HC5S1,
        dtype=np.float64,
    ),
    rul_cap=np.array(
        RUL_CAP_628HC5S1,
        dtype=np.float64,
    ),
)

# ---------------------------------------------------------------
# 4. Freeze human-readable tables
# ---------------------------------------------------------------
FD002_TEST_SCORING_TABLE_628HC5S1.to_csv(
    TABLE_CSV_PATH_628HC5F3A,
    index=False,
    float_format="%.17g",
)

FD002_TEST_SCORE_SUMMARY_628HC5S1.to_csv(
    SUMMARY_CSV_PATH_628HC5F3A,
    index=False,
    float_format="%.17g",
)

# ---------------------------------------------------------------
# 5. File hashes + metadata
# ---------------------------------------------------------------
def sha256_file_628hc5f3a(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

npz_hash_628hc5f3a = sha256_file_628hc5f3a(
    NPZ_PATH_628HC5F3A
)

table_csv_hash_628hc5f3a = sha256_file_628hc5f3a(
    TABLE_CSV_PATH_628HC5F3A
)

summary_csv_hash_628hc5f3a = sha256_file_628hc5f3a(
    SUMMARY_CSV_PATH_628HC5F3A
)

metadata_628hc5f3a = {
    "step": "6.28H-C5-F3-A",
    "dataset": "FD002",
    "status": "official_TEST_scored_and_frozen",
    "test_endpoints": 259,
    "evaluation_target": "piecewise-capped RUL",
    "rul_cap": float(RUL_CAP_628HC5S1),
    "reliability_architecture": "uncertainty_only",
    "operating_policy": "uncertainty_only_fixed_threshold",
    "frozen_nominal_coverage": 0.8,
    "frozen_exact_risk_threshold":
        float(THRESHOLD_628HC5S1),
    "accepted_test_endpoints":
        int(np.sum(accept_628hc5s1)),
    "abstained_test_endpoints":
        int(np.sum(abstain_628hc5s1)),
    "realized_test_coverage":
        float(np.mean(accept_628hc5s1)),
    "full_test_mae":
        float(m_all_628hc5s1["mae"]),
    "accepted_test_mae":
        float(m_acc_628hc5s1["mae"]),
    "abstained_test_mae":
        float(m_abs_628hc5s1["mae"]),
    "full_test_rmse":
        float(m_all_628hc5s1["rmse"]),
    "accepted_test_rmse":
        float(m_acc_628hc5s1["rmse"]),
    "abstained_test_rmse":
        float(m_abs_628hc5s1["rmse"]),
    "full_nasa_score_total":
        float(m_all_628hc5s1["nasa_score_total"]),
    "accepted_nasa_score_total":
        float(m_acc_628hc5s1["nasa_score_total"]),
    "abstained_nasa_score_total":
        float(m_abs_628hc5s1["nasa_score_total"]),
    "risk_abs_error_spearman":
        float(risk_error_spearman_628hc5s1),
    "prescoring_contract_sha256":
        PRESCORING_SHA_628HC5S1,
    "array_sha256":
        array_hashes_628hc5f3a,
    "file_sha256": {
        "fd002_c5_official_test_scoring.npz":
            npz_hash_628hc5f3a,
        "fd002_c5_official_test_scoring.csv":
            table_csv_hash_628hc5f3a,
        "fd002_c5_official_test_score_summary.csv":
            summary_csv_hash_628hc5f3a,
    },
    "threshold_tuned_after_test": False,
    "threshold_modified_after_test": False,
    "model_executed_in_scoring_step": False,
    "model_retrained_after_test": False,
}

META_PATH_628HC5F3A.write_text(
    json.dumps(
        metadata_628hc5f3a,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

metadata_hash_628hc5f3a = sha256_file_628hc5f3a(
    META_PATH_628HC5F3A
)

# ---------------------------------------------------------------
# 6. Immediate same-session write audit
# ---------------------------------------------------------------
write_audit_628hc5f3a = pd.DataFrame(
    [
        ("npz_exists", NPZ_PATH_628HC5F3A.is_file()),
        ("scoring_csv_exists", TABLE_CSV_PATH_628HC5F3A.is_file()),
        ("summary_csv_exists", SUMMARY_CSV_PATH_628HC5F3A.is_file()),
        ("metadata_exists", META_PATH_628HC5F3A.is_file()),
        ("npz_nonempty", NPZ_PATH_628HC5F3A.stat().st_size > 0),
        (
            "scoring_csv_nonempty",
            TABLE_CSV_PATH_628HC5F3A.stat().st_size > 0,
        ),
        (
            "summary_csv_nonempty",
            SUMMARY_CSV_PATH_628HC5F3A.stat().st_size > 0,
        ),
        (
            "metadata_nonempty",
            META_PATH_628HC5F3A.stat().st_size > 0,
        ),
    ],
    columns=["check", "passed"],
)

print("\n--- C5-F3-A WRITE AUDIT ---")
display(write_audit_628hc5f3a)

if not bool(write_audit_628hc5f3a["passed"].all()):
    raise RuntimeError(
        "C5-F3-A write audit failed."
    )

# ---------------------------------------------------------------
# 7. Status
# ---------------------------------------------------------------
print("\n--- WRITTEN ARTIFACTS ---")

for p in [
    NPZ_PATH_628HC5F3A,
    TABLE_CSV_PATH_628HC5F3A,
    SUMMARY_CSV_PATH_628HC5F3A,
    META_PATH_628HC5F3A,
]:
    print(
        p.name,
        "| exists =",
        p.is_file(),
        "| bytes =",
        p.stat().st_size,
    )

print("\n--- ARRAY SHA-256 ---")
for name, value in array_hashes_628hc5f3a.items():
    print(f"{name:28s}: {value}")

print("\n--- FILE SHA-256 ---")
print("NPZ         :", npz_hash_628hc5f3a)
print("scoring CSV :", table_csv_hash_628hc5f3a)
print("summary CSV :", summary_csv_hash_628hc5f3a)
print("metadata    :", metadata_hash_628hc5f3a)

print("\n--- F3-A STATUS ---")
print("All files written correctly   : True")
print("Official TEST labels read      : True")
print("TEST scored                    : True")
print("Threshold tuned or modified    : False")
print("Model executed                 : False")
print("Model retrained                : False")
print("Files written                  : True")
print("Independent reload verified    : False")

print(
    "\nNext required step: "
    "C5-F3-B independent reload and exact verification."
)

print(
    "\n=== END STEP 6.28H-C5-F3-A ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-F3-B
# INDEPENDENT RELOAD + EXACT VERIFICATION OF OFFICIAL TEST SCORING
# ================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F3-B — INDEPENDENT RELOAD + EXACT TEST-SCORING VERIFICATION")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Reconstruct artifact paths directly from disk contract
# ---------------------------------------------------------------
FREEZE_DIR_628HC5F3B = Path(
    r"C:\RUL_Reliability_Research\06_results\step6_reliability_signals\fd002_c5"
)

NPZ_PATH_628HC5F3B = (
    FREEZE_DIR_628HC5F3B
    / "fd002_c5_official_test_scoring.npz"
)

TABLE_CSV_PATH_628HC5F3B = (
    FREEZE_DIR_628HC5F3B
    / "fd002_c5_official_test_scoring.csv"
)

SUMMARY_CSV_PATH_628HC5F3B = (
    FREEZE_DIR_628HC5F3B
    / "fd002_c5_official_test_score_summary.csv"
)

META_PATH_628HC5F3B = (
    FREEZE_DIR_628HC5F3B
    / "fd002_c5_official_test_scoring_metadata.json"
)

files_present_628hc5f3b = all(
    p.is_file()
    for p in [
        NPZ_PATH_628HC5F3B,
        TABLE_CSV_PATH_628HC5F3B,
        SUMMARY_CSV_PATH_628HC5F3B,
        META_PATH_628HC5F3B,
    ]
)

if not files_present_628hc5f3b:
    raise RuntimeError(
        "C5-F3-B cannot run: one or more frozen F3-A artifacts are missing."
    )

# ---------------------------------------------------------------
# 2. Independent reload
# ---------------------------------------------------------------
with np.load(
    NPZ_PATH_628HC5F3B,
    allow_pickle=False,
) as z:
    npz_keys_628hc5f3b = tuple(sorted(z.files))

    endpoint_disk_628hc5f3b = np.array(
        z["endpoint_position_1based"],
        copy=True,
    )
    y_raw_disk_628hc5f3b = np.array(
        z["official_rul_raw"],
        copy=True,
    )
    y_eval_disk_628hc5f3b = np.array(
        z["evaluation_rul_capped_125"],
        copy=True,
    )
    pred_disk_628hc5f3b = np.array(
        z["point_prediction"],
        copy=True,
    )
    risk_disk_628hc5f3b = np.array(
        z["reliability_risk"],
        copy=True,
    )
    accept_disk_628hc5f3b = np.array(
        z["accept_mask"],
        copy=True,
    )
    abstain_disk_628hc5f3b = np.array(
        z["abstain_mask"],
        copy=True,
    )
    signed_error_disk_628hc5f3b = np.array(
        z["signed_error"],
        copy=True,
    )
    abs_error_disk_628hc5f3b = np.array(
        z["absolute_error"],
        copy=True,
    )

    threshold_disk_628hc5f3b = float(
        np.asarray(z["frozen_risk_threshold"]).item()
    )
    cap_disk_628hc5f3b = float(
        np.asarray(z["rul_cap"]).item()
    )

table_disk_628hc5f3b = pd.read_csv(
    TABLE_CSV_PATH_628HC5F3B
)

summary_disk_628hc5f3b = pd.read_csv(
    SUMMARY_CSV_PATH_628HC5F3B
)

metadata_disk_628hc5f3b = json.loads(
    META_PATH_628HC5F3B.read_text(
        encoding="utf-8"
    )
)

# ---------------------------------------------------------------
# 3. Hash helpers
# ---------------------------------------------------------------
def sha256_array_628hc5f3b(arr):
    a = np.ascontiguousarray(arr)
    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def sha256_file_628hc5f3b(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ---------------------------------------------------------------
# 4. Expected NPZ schema
# ---------------------------------------------------------------
expected_npz_keys_628hc5f3b = tuple(
    sorted(
        [
            "endpoint_position_1based",
            "official_rul_raw",
            "evaluation_rul_capped_125",
            "point_prediction",
            "reliability_risk",
            "accept_mask",
            "abstain_mask",
            "signed_error",
            "absolute_error",
            "frozen_risk_threshold",
            "rul_cap",
        ]
    )
)

# ---------------------------------------------------------------
# 5. File-hash verification against frozen metadata
# ---------------------------------------------------------------
npz_file_hash_disk_628hc5f3b = sha256_file_628hc5f3b(
    NPZ_PATH_628HC5F3B
)

table_file_hash_disk_628hc5f3b = sha256_file_628hc5f3b(
    TABLE_CSV_PATH_628HC5F3B
)

summary_file_hash_disk_628hc5f3b = sha256_file_628hc5f3b(
    SUMMARY_CSV_PATH_628HC5F3B
)

metadata_file_hash_disk_628hc5f3b = sha256_file_628hc5f3b(
    META_PATH_628HC5F3B
)

frozen_file_hashes_628hc5f3b = metadata_disk_628hc5f3b[
    "file_sha256"
]

# ---------------------------------------------------------------
# 6. Array-hash verification against frozen metadata
# ---------------------------------------------------------------
array_hashes_disk_628hc5f3b = {
    "endpoint_position_1based":
        sha256_array_628hc5f3b(
            endpoint_disk_628hc5f3b
        ),
    "official_rul_raw":
        sha256_array_628hc5f3b(
            y_raw_disk_628hc5f3b
        ),
    "evaluation_rul_capped_125":
        sha256_array_628hc5f3b(
            y_eval_disk_628hc5f3b
        ),
    "point_prediction":
        sha256_array_628hc5f3b(
            pred_disk_628hc5f3b
        ),
    "reliability_risk":
        sha256_array_628hc5f3b(
            risk_disk_628hc5f3b
        ),
    "accept_mask":
        sha256_array_628hc5f3b(
            accept_disk_628hc5f3b
        ),
    "abstain_mask":
        sha256_array_628hc5f3b(
            abstain_disk_628hc5f3b
        ),
    "signed_error":
        sha256_array_628hc5f3b(
            signed_error_disk_628hc5f3b
        ),
    "absolute_error":
        sha256_array_628hc5f3b(
            abs_error_disk_628hc5f3b
        ),
}

array_hashes_exact_628hc5f3b = (
    array_hashes_disk_628hc5f3b
    == metadata_disk_628hc5f3b[
        "array_sha256"
    ]
)

# ---------------------------------------------------------------
# 7. Exact binary reload vs authoritative in-memory S1 objects
# ---------------------------------------------------------------
endpoint_expected_628hc5f3b = np.arange(
    1,
    260,
    dtype=np.int64,
)

binary_exact_checks_628hc5f3b = {
    "endpoint":
        np.array_equal(
            endpoint_disk_628hc5f3b,
            endpoint_expected_628hc5f3b,
        ),

    "official_rul_raw":
        np.array_equal(
            y_raw_disk_628hc5f3b,
            np.asarray(
                y_raw_628hc5s1,
                dtype=np.float64,
            ),
        ),

    "evaluation_rul":
        np.array_equal(
            y_eval_disk_628hc5f3b,
            np.asarray(
                y_eval_628hc5s1,
                dtype=np.float64,
            ),
        ),

    "point_prediction":
        np.array_equal(
            pred_disk_628hc5f3b,
            np.asarray(
                pred_628hc5s1,
                dtype=np.float64,
            ),
        ),

    "reliability_risk":
        np.array_equal(
            risk_disk_628hc5f3b,
            np.asarray(
                risk_628hc5s1,
                dtype=np.float64,
            ),
        ),

    "accept_mask":
        np.array_equal(
            accept_disk_628hc5f3b,
            np.asarray(
                accept_628hc5s1,
                dtype=np.bool_,
            ),
        ),

    "abstain_mask":
        np.array_equal(
            abstain_disk_628hc5f3b,
            np.asarray(
                abstain_628hc5s1,
                dtype=np.bool_,
            ),
        ),

    "signed_error":
        np.array_equal(
            signed_error_disk_628hc5f3b,
            np.asarray(
                signed_error_628hc5s1,
                dtype=np.float64,
            ),
        ),

    "absolute_error":
        np.array_equal(
            abs_error_disk_628hc5f3b,
            np.asarray(
                abs_error_628hc5s1,
                dtype=np.float64,
            ),
        ),
}

binary_reload_exact_628hc5f3b = bool(
    all(
        binary_exact_checks_628hc5f3b.values()
    )
)

# ---------------------------------------------------------------
# 8. CSV semantic round-trip verification
#
# NPZ is authoritative binary storage.
# CSV must preserve numerical meaning to <= 1e-12.
# ---------------------------------------------------------------
def dataframe_semantic_close_628hc5f3b(
    reloaded,
    reference,
    atol=1e-12,
):
    if list(reloaded.columns) != list(
        reference.columns
    ):
        return False, np.inf

    if reloaded.shape != reference.shape:
        return False, np.inf

    max_abs_diff = 0.0

    for col in reference.columns:
        ref_col = reference[col]
        new_col = reloaded[col]

        ref_numeric = (
            pd.api.types.is_numeric_dtype(
                ref_col
            )
            and not pd.api.types.is_bool_dtype(
                ref_col
            )
        )

        new_numeric = (
            pd.api.types.is_numeric_dtype(
                new_col
            )
            and not pd.api.types.is_bool_dtype(
                new_col
            )
        )

        if ref_numeric and new_numeric:
            ref_values = ref_col.to_numpy(
                dtype=np.float64
            )
            new_values = new_col.to_numpy(
                dtype=np.float64
            )

            if not np.allclose(
                new_values,
                ref_values,
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            ):
                return (
                    False,
                    float(
                        np.nanmax(
                            np.abs(
                                new_values
                                - ref_values
                            )
                        )
                    ),
                )

            if ref_values.size:
                col_diff = float(
                    np.nanmax(
                        np.abs(
                            new_values
                            - ref_values
                        )
                    )
                )

                max_abs_diff = max(
                    max_abs_diff,
                    col_diff,
                )

        else:
            ref_values = ref_col.to_numpy()
            new_values = new_col.to_numpy()

            if not np.array_equal(
                new_values,
                ref_values,
                equal_nan=True,
            ):
                return False, np.inf

    return True, max_abs_diff


(
    table_semantic_ok_628hc5f3b,
    table_maxdiff_628hc5f3b,
) = dataframe_semantic_close_628hc5f3b(
    table_disk_628hc5f3b,
    FD002_TEST_SCORING_TABLE_628HC5S1,
    atol=1e-12,
)

(
    summary_semantic_ok_628hc5f3b,
    summary_maxdiff_628hc5f3b,
) = dataframe_semantic_close_628hc5f3b(
    summary_disk_628hc5f3b,
    FD002_TEST_SCORE_SUMMARY_628HC5S1,
    atol=1e-12,
)

# ---------------------------------------------------------------
# 9. Frozen metadata contract
# ---------------------------------------------------------------
metadata_contract_exact_628hc5f3b = bool(
    metadata_disk_628hc5f3b["step"]
    == "6.28H-C5-F3-A"

    and metadata_disk_628hc5f3b["dataset"]
    == "FD002"

    and metadata_disk_628hc5f3b["status"]
    == "official_TEST_scored_and_frozen"

    and metadata_disk_628hc5f3b[
        "test_endpoints"
    ] == 259

    and metadata_disk_628hc5f3b[
        "evaluation_target"
    ] == "piecewise-capped RUL"

    and metadata_disk_628hc5f3b[
        "rul_cap"
    ] == float(
        RUL_CAP_628HC5S1
    )

    and metadata_disk_628hc5f3b[
        "reliability_architecture"
    ] == "uncertainty_only"

    and metadata_disk_628hc5f3b[
        "operating_policy"
    ] == "uncertainty_only_fixed_threshold"

    and metadata_disk_628hc5f3b[
        "frozen_nominal_coverage"
    ] == 0.8

    and metadata_disk_628hc5f3b[
        "frozen_exact_risk_threshold"
    ] == float(
        THRESHOLD_628HC5S1
    )

    and metadata_disk_628hc5f3b[
        "accepted_test_endpoints"
    ] == 211

    and metadata_disk_628hc5f3b[
        "abstained_test_endpoints"
    ] == 48

    and metadata_disk_628hc5f3b[
        "realized_test_coverage"
    ] == float(
        np.mean(
            accept_628hc5s1
        )
    )

    and metadata_disk_628hc5f3b[
        "prescoring_contract_sha256"
    ] == PRESCORING_SHA_628HC5S1

    and metadata_disk_628hc5f3b[
        "threshold_tuned_after_test"
    ] is False

    and metadata_disk_628hc5f3b[
        "threshold_modified_after_test"
    ] is False

    and metadata_disk_628hc5f3b[
        "model_executed_in_scoring_step"
    ] is False

    and metadata_disk_628hc5f3b[
        "model_retrained_after_test"
    ] is False
)

# ---------------------------------------------------------------
# 10. Decision contract from independently reloaded arrays
# ---------------------------------------------------------------
decision_contract_exact_628hc5f3b = bool(
    threshold_disk_628hc5f3b
    == float(
        THRESHOLD_628HC5S1
    )

    and cap_disk_628hc5f3b
    == float(
        RUL_CAP_628HC5S1
    )

    and accept_disk_628hc5f3b.dtype
    == np.bool_

    and abstain_disk_628hc5f3b.dtype
    == np.bool_

    and np.array_equal(
        abstain_disk_628hc5f3b,
        ~accept_disk_628hc5f3b,
    )

    and np.array_equal(
        accept_disk_628hc5f3b,
        risk_disk_628hc5f3b
        <= threshold_disk_628hc5f3b,
    )

    and int(
        accept_disk_628hc5f3b.sum()
    ) == 211

    and int(
        abstain_disk_628hc5f3b.sum()
    ) == 48
)

# ---------------------------------------------------------------
# 11. Final independent reload audit
# ---------------------------------------------------------------
audit_628hc5f3b = pd.DataFrame(
    [
        (
            "all_four_F3A_files_present",
            files_present_628hc5f3b,
        ),

        (
            "npz_keys_exact",
            npz_keys_628hc5f3b
            == expected_npz_keys_628hc5f3b,
        ),

        (
            "all_binary_shapes_259",
            all(
                np.asarray(x).shape == (259,)
                for x in [
                    endpoint_disk_628hc5f3b,
                    y_raw_disk_628hc5f3b,
                    y_eval_disk_628hc5f3b,
                    pred_disk_628hc5f3b,
                    risk_disk_628hc5f3b,
                    accept_disk_628hc5f3b,
                    abstain_disk_628hc5f3b,
                    signed_error_disk_628hc5f3b,
                    abs_error_disk_628hc5f3b,
                ]
            ),
        ),

        (
            "all_binary_values_finite",
            all(
                np.isfinite(
                    np.asarray(
                        x,
                        dtype=np.float64,
                    )
                ).all()
                for x in [
                    y_raw_disk_628hc5f3b,
                    y_eval_disk_628hc5f3b,
                    pred_disk_628hc5f3b,
                    risk_disk_628hc5f3b,
                    signed_error_disk_628hc5f3b,
                    abs_error_disk_628hc5f3b,
                ]
            ),
        ),

        (
            "risk_within_0_1",
            bool(
                np.all(
                    (
                        risk_disk_628hc5f3b
                        >= 0.0
                    )
                    &
                    (
                        risk_disk_628hc5f3b
                        <= 1.0
                    )
                )
            ),
        ),

        (
            "npz_file_hash_exact",
            npz_file_hash_disk_628hc5f3b
            == frozen_file_hashes_628hc5f3b[
                "fd002_c5_official_test_scoring.npz"
            ],
        ),

        (
            "scoring_csv_file_hash_exact",
            table_file_hash_disk_628hc5f3b
            == frozen_file_hashes_628hc5f3b[
                "fd002_c5_official_test_scoring.csv"
            ],
        ),

        (
            "summary_csv_file_hash_exact",
            summary_file_hash_disk_628hc5f3b
            == frozen_file_hashes_628hc5f3b[
                "fd002_c5_official_test_score_summary.csv"
            ],
        ),

        (
            "all_array_hashes_exact",
            array_hashes_exact_628hc5f3b,
        ),

        (
            "binary_reload_exact_vs_memory",
            binary_reload_exact_628hc5f3b,
        ),

        (
            "scoring_csv_semantic_roundtrip_1e12",
            table_semantic_ok_628hc5f3b,
        ),

        (
            "summary_csv_semantic_roundtrip_1e12",
            summary_semantic_ok_628hc5f3b,
        ),

        (
            "metadata_contract_exact",
            metadata_contract_exact_628hc5f3b,
        ),

        (
            "decision_contract_exact",
            decision_contract_exact_628hc5f3b,
        ),

        (
            "prescoring_contract_still_exact",
            PRESCORING_SHA_628HC5S1
            == FD002_TEST_PRESCORING_SHA256_628HC5A5,
        ),

        (
            "official_test_label_marker_true",
            FD002_OFFICIAL_TEST_LABELS_READ_628A
            is True,
        ),

        (
            "no_test_file_read_in_this_step",
            True,
        ),

        (
            "no_test_rescoring_in_this_step",
            True,
        ),

        (
            "no_threshold_tuning",
            True,
        ),

        (
            "no_threshold_modification",
            True,
        ),

        (
            "no_model_execution",
            True,
        ),

        (
            "no_model_retraining",
            True,
        ),

        (
            "no_files_modified",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C5-F3-B INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc5f3b
)

all_checks_passed_628hc5f3b = bool(
    audit_628hc5f3b[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5f3b:
    raise RuntimeError(
        "C5-F3-B independent reload verification failed."
    )

# ---------------------------------------------------------------
# 12. Final summary
# ---------------------------------------------------------------
print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C5-F3-B FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Canonical TEST risk shape          :",
    risk_disk_628hc5f3b.shape,
)

print(
    "Canonical TEST prediction shape    :",
    pred_disk_628hc5f3b.shape,
)

print(
    "Frozen exact threshold             :",
    threshold_disk_628hc5f3b,
)

print(
    "Accepted / abstained endpoints     :",
    int(
        accept_disk_628hc5f3b.sum()
    ),
    "/",
    int(
        abstain_disk_628hc5f3b.sum()
    ),
)

print(
    "Realized TEST coverage             :",
    float(
        accept_disk_628hc5f3b.mean()
    ),
)

print(
    "Binary artifact exact vs memory    :",
    binary_reload_exact_628hc5f3b,
)

print(
    "All stored array hashes exact      :",
    array_hashes_exact_628hc5f3b,
)

print(
    "NPZ file hash exact                :",
    npz_file_hash_disk_628hc5f3b
    == frozen_file_hashes_628hc5f3b[
        "fd002_c5_official_test_scoring.npz"
    ],
)

print(
    "Scoring CSV hash exact             :",
    table_file_hash_disk_628hc5f3b
    == frozen_file_hashes_628hc5f3b[
        "fd002_c5_official_test_scoring.csv"
    ],
)

print(
    "Summary CSV hash exact             :",
    summary_file_hash_disk_628hc5f3b
    == frozen_file_hashes_628hc5f3b[
        "fd002_c5_official_test_score_summary.csv"
    ],
)

print(
    "Scoring CSV max numerical diff     :",
    table_maxdiff_628hc5f3b,
)

print(
    "Summary CSV max numerical diff     :",
    summary_maxdiff_628hc5f3b,
)

print(
    "Metadata contract exact            :",
    metadata_contract_exact_628hc5f3b,
)

print(
    "Decision contract exact            :",
    decision_contract_exact_628hc5f3b,
)

print(
    "Metadata file SHA-256              :",
    metadata_file_hash_disk_628hc5f3b,
)

print(
    "Independent reload verified        :",
    all_checks_passed_628hc5f3b,
)

print(
    "Official TEST label file read here :",
    False,
)

print(
    "TEST rescored in this step         :",
    False,
)

print(
    "Threshold tuned or modified        :",
    False,
)

print(
    "Model executed                     :",
    False,
)

print(
    "Model retrained                    :",
    False,
)

print(
    "Files modified                     :",
    False,
)

print(
    "\nFD002 official TEST scoring is now "
    "fully frozen and independently verified."
)

print(
    "\n=== END STEP 6.28H-C5-F3-B ==="
)

In [ ]:
# ================================================================
# STEP 6.28H-C5-F3-B-R1
# REPAIR: CSV MIXED-DTYPE SEMANTIC ROUND-TRIP + COMPLETE RELOAD AUDIT
# ================================================================

import numpy as np
import pandas as pd

print("=" * 94)
print("STEP 6.28H-C5-F3-B-R1 — CSV MIXED-DTYPE ROUND-TRIP REPAIR")
print("=" * 94)

# ---------------------------------------------------------------
# 1. Robust mixed-dtype DataFrame semantic comparison
# ---------------------------------------------------------------
def dataframe_semantic_close_628hc5f3br1(reloaded, reference, atol=1e-12):

    if list(reloaded.columns) != list(reference.columns):
        return False, np.inf

    if reloaded.shape != reference.shape:
        return False, np.inf

    max_abs_diff = 0.0

    for col in reference.columns:

        ref_col = reference[col]
        new_col = reloaded[col]

        ref_numeric = (
            pd.api.types.is_numeric_dtype(ref_col)
            and not pd.api.types.is_bool_dtype(ref_col)
        )

        new_numeric = (
            pd.api.types.is_numeric_dtype(new_col)
            and not pd.api.types.is_bool_dtype(new_col)
        )

        # -------------------------------------------------------
        # Numeric columns
        # -------------------------------------------------------
        if ref_numeric and new_numeric:

            ref_values = ref_col.to_numpy(dtype=np.float64)
            new_values = new_col.to_numpy(dtype=np.float64)

            close = np.isclose(
                new_values,
                ref_values,
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            )

            if not bool(np.all(close)):

                finite_pair = (
                    np.isfinite(new_values)
                    & np.isfinite(ref_values)
                )

                if np.any(finite_pair):
                    diff = float(
                        np.max(
                            np.abs(
                                new_values[finite_pair]
                                - ref_values[finite_pair]
                            )
                        )
                    )
                else:
                    diff = np.inf

                return False, diff

            finite_pair = (
                np.isfinite(new_values)
                & np.isfinite(ref_values)
            )

            if np.any(finite_pair):

                col_diff = float(
                    np.max(
                        np.abs(
                            new_values[finite_pair]
                            - ref_values[finite_pair]
                        )
                    )
                )

                max_abs_diff = max(
                    max_abs_diff,
                    col_diff,
                )

        # -------------------------------------------------------
        # Strings / objects / booleans / categories
        # -------------------------------------------------------
        else:

            ref_text = (
                ref_col
                .astype("string")
                .reset_index(drop=True)
            )

            new_text = (
                new_col
                .astype("string")
                .reset_index(drop=True)
            )

            if not bool(
                ref_text.equals(new_text)
            ):
                return False, np.inf

    return True, max_abs_diff


# ---------------------------------------------------------------
# 2. Re-run ONLY the two CSV semantic checks
# ---------------------------------------------------------------
(
    table_semantic_ok_628hc5f3b,
    table_maxdiff_628hc5f3b,
) = dataframe_semantic_close_628hc5f3br1(
    table_disk_628hc5f3b,
    FD002_TEST_SCORING_TABLE_628HC5S1,
    atol=1e-12,
)

(
    summary_semantic_ok_628hc5f3b,
    summary_maxdiff_628hc5f3b,
) = dataframe_semantic_close_628hc5f3br1(
    summary_disk_628hc5f3b,
    FD002_TEST_SCORE_SUMMARY_628HC5S1,
    atol=1e-12,
)

print("\n--- CSV ROUND-TRIP DIAGNOSTIC ---")

print(
    "Scoring CSV semantic match :",
    table_semantic_ok_628hc5f3b,
)

print(
    "Scoring CSV max abs diff   :",
    table_maxdiff_628hc5f3b,
)

print(
    "Summary CSV semantic match :",
    summary_semantic_ok_628hc5f3b,
)

print(
    "Summary CSV max abs diff   :",
    summary_maxdiff_628hc5f3b,
)


# ---------------------------------------------------------------
# 3. Frozen metadata contract
# ---------------------------------------------------------------
metadata_contract_exact_628hc5f3b = bool(

    metadata_disk_628hc5f3b["step"]
    == "6.28H-C5-F3-A"

    and metadata_disk_628hc5f3b["dataset"]
    == "FD002"

    and metadata_disk_628hc5f3b["status"]
    == "official_TEST_scored_and_frozen"

    and metadata_disk_628hc5f3b[
        "test_endpoints"
    ] == 259

    and metadata_disk_628hc5f3b[
        "evaluation_target"
    ] == "piecewise-capped RUL"

    and metadata_disk_628hc5f3b[
        "rul_cap"
    ] == float(RUL_CAP_628HC5S1)

    and metadata_disk_628hc5f3b[
        "reliability_architecture"
    ] == "uncertainty_only"

    and metadata_disk_628hc5f3b[
        "operating_policy"
    ] == "uncertainty_only_fixed_threshold"

    and metadata_disk_628hc5f3b[
        "frozen_nominal_coverage"
    ] == 0.8

    and metadata_disk_628hc5f3b[
        "frozen_exact_risk_threshold"
    ] == float(THRESHOLD_628HC5S1)

    and metadata_disk_628hc5f3b[
        "accepted_test_endpoints"
    ] == 211

    and metadata_disk_628hc5f3b[
        "abstained_test_endpoints"
    ] == 48

    and metadata_disk_628hc5f3b[
        "realized_test_coverage"
    ] == float(
        np.mean(
            accept_628hc5s1
        )
    )

    and metadata_disk_628hc5f3b[
        "prescoring_contract_sha256"
    ] == PRESCORING_SHA_628HC5S1

    and metadata_disk_628hc5f3b[
        "threshold_tuned_after_test"
    ] is False

    and metadata_disk_628hc5f3b[
        "threshold_modified_after_test"
    ] is False

    and metadata_disk_628hc5f3b[
        "model_executed_in_scoring_step"
    ] is False

    and metadata_disk_628hc5f3b[
        "model_retrained_after_test"
    ] is False
)


# ---------------------------------------------------------------
# 4. Independently reloaded decision contract
# ---------------------------------------------------------------
decision_contract_exact_628hc5f3b = bool(

    threshold_disk_628hc5f3b
    == float(THRESHOLD_628HC5S1)

    and cap_disk_628hc5f3b
    == float(RUL_CAP_628HC5S1)

    and accept_disk_628hc5f3b.dtype
    == np.bool_

    and abstain_disk_628hc5f3b.dtype
    == np.bool_

    and np.array_equal(
        abstain_disk_628hc5f3b,
        ~accept_disk_628hc5f3b,
    )

    and np.array_equal(
        accept_disk_628hc5f3b,
        risk_disk_628hc5f3b
        <= threshold_disk_628hc5f3b,
    )

    and int(
        accept_disk_628hc5f3b.sum()
    ) == 211

    and int(
        abstain_disk_628hc5f3b.sum()
    ) == 48
)


# ---------------------------------------------------------------
# 5. Final independent reload audit
# ---------------------------------------------------------------
audit_628hc5f3br1 = pd.DataFrame(
    [
        (
            "all_four_F3A_files_present",
            bool(files_present_628hc5f3b),
        ),

        (
            "npz_keys_exact",
            bool(
                npz_keys_628hc5f3b
                == expected_npz_keys_628hc5f3b
            ),
        ),

        (
            "all_binary_shapes_259",
            bool(
                all(
                    np.asarray(x).shape
                    == (259,)
                    for x in [
                        endpoint_disk_628hc5f3b,
                        y_raw_disk_628hc5f3b,
                        y_eval_disk_628hc5f3b,
                        pred_disk_628hc5f3b,
                        risk_disk_628hc5f3b,
                        accept_disk_628hc5f3b,
                        abstain_disk_628hc5f3b,
                        signed_error_disk_628hc5f3b,
                        abs_error_disk_628hc5f3b,
                    ]
                )
            ),
        ),

        (
            "all_binary_values_finite",
            bool(
                all(
                    np.isfinite(
                        np.asarray(
                            x,
                            dtype=np.float64,
                        )
                    ).all()
                    for x in [
                        y_raw_disk_628hc5f3b,
                        y_eval_disk_628hc5f3b,
                        pred_disk_628hc5f3b,
                        risk_disk_628hc5f3b,
                        signed_error_disk_628hc5f3b,
                        abs_error_disk_628hc5f3b,
                    ]
                )
            ),
        ),

        (
            "risk_within_0_1",
            bool(
                np.all(
                    (
                        risk_disk_628hc5f3b >= 0.0
                    )
                    &
                    (
                        risk_disk_628hc5f3b <= 1.0
                    )
                )
            ),
        ),

        (
            "npz_file_hash_exact",
            bool(
                npz_file_hash_disk_628hc5f3b
                ==
                frozen_file_hashes_628hc5f3b[
                    "fd002_c5_official_test_scoring.npz"
                ]
            ),
        ),

        (
            "scoring_csv_file_hash_exact",
            bool(
                table_file_hash_disk_628hc5f3b
                ==
                frozen_file_hashes_628hc5f3b[
                    "fd002_c5_official_test_scoring.csv"
                ]
            ),
        ),

        (
            "summary_csv_file_hash_exact",
            bool(
                summary_file_hash_disk_628hc5f3b
                ==
                frozen_file_hashes_628hc5f3b[
                    "fd002_c5_official_test_score_summary.csv"
                ]
            ),
        ),

        (
            "all_array_hashes_exact",
            bool(
                array_hashes_exact_628hc5f3b
            ),
        ),

        (
            "binary_reload_exact_vs_memory",
            bool(
                binary_reload_exact_628hc5f3b
            ),
        ),

        (
            "scoring_csv_semantic_roundtrip_1e12",
            bool(
                table_semantic_ok_628hc5f3b
            ),
        ),

        (
            "summary_csv_semantic_roundtrip_1e12",
            bool(
                summary_semantic_ok_628hc5f3b
            ),
        ),

        (
            "metadata_contract_exact",
            bool(
                metadata_contract_exact_628hc5f3b
            ),
        ),

        (
            "decision_contract_exact",
            bool(
                decision_contract_exact_628hc5f3b
            ),
        ),

        (
            "prescoring_contract_still_exact",
            bool(
                PRESCORING_SHA_628HC5S1
                ==
                FD002_TEST_PRESCORING_SHA256_628HC5A5
            ),
        ),

        (
            "official_test_label_marker_true",
            bool(
                FD002_OFFICIAL_TEST_LABELS_READ_628A
                is True
            ),
        ),

        (
            "no_test_file_read_in_this_step",
            True,
        ),

        (
            "no_test_rescoring_in_this_step",
            True,
        ),

        (
            "no_threshold_tuning",
            True,
        ),

        (
            "no_threshold_modification",
            True,
        ),

        (
            "no_model_execution",
            True,
        ),

        (
            "no_model_retraining",
            True,
        ),

        (
            "no_files_modified",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C5-F3-B-R1 INDEPENDENT RELOAD AUDIT ---"
)

display(
    audit_628hc5f3br1
)

all_checks_passed_628hc5f3br1 = bool(
    audit_628hc5f3br1[
        "passed"
    ].all()
)

if not all_checks_passed_628hc5f3br1:
    raise RuntimeError(
        "C5-F3-B-R1 independent reload "
        "verification did not fully pass."
    )


# ---------------------------------------------------------------
# 6. Final summary
# ---------------------------------------------------------------
print(
    "\n"
    + "=" * 94
)

print(
    "STEP 6.28H-C5-F3-B-R1 FINAL SUMMARY"
)

print(
    "=" * 94
)

print(
    "Canonical TEST risk shape          :",
    risk_disk_628hc5f3b.shape,
)

print(
    "Canonical TEST prediction shape    :",
    pred_disk_628hc5f3b.shape,
)

print(
    "Frozen exact threshold             :",
    threshold_disk_628hc5f3b,
)

print(
    "Accepted / abstained endpoints     :",
    int(
        accept_disk_628hc5f3b.sum()
    ),
    "/",
    int(
        abstain_disk_628hc5f3b.sum()
    ),
)

print(
    "Realized TEST coverage             :",
    float(
        accept_disk_628hc5f3b.mean()
    ),
)

print(
    "Binary artifact exact vs memory    :",
    binary_reload_exact_628hc5f3b,
)

print(
    "All stored array hashes exact      :",
    array_hashes_exact_628hc5f3b,
)

print(
    "Scoring CSV semantic round-trip    :",
    table_semantic_ok_628hc5f3b,
)

print(
    "Scoring CSV max numerical diff     :",
    table_maxdiff_628hc5f3b,
)

print(
    "Summary CSV semantic round-trip    :",
    summary_semantic_ok_628hc5f3b,
)

print(
    "Summary CSV max numerical diff     :",
    summary_maxdiff_628hc5f3b,
)

print(
    "Metadata contract exact            :",
    metadata_contract_exact_628hc5f3b,
)

print(
    "Decision contract exact            :",
    decision_contract_exact_628hc5f3b,
)

print(
    "Independent reload verified        :",
    all_checks_passed_628hc5f3br1,
)

print(
    "Official TEST file reread here     :",
    False,
)

print(
    "TEST rescored in this step         :",
    False,
)

print(
    "Threshold tuned or modified        :",
    False,
)

print(
    "Model executed                     :",
    False,
)

print(
    "Model retrained                    :",
    False,
)

print(
    "Files modified                     :",
    False,
)

print(
    "\n=== END STEP 6.28H-C5-F3-B-R1 ==="
)

In [ ]:
# =============================================================================
# STEP 6.28H-C5-M0 — MASTER FD002 C5 SUMMARY
# Post-scoring consolidation only: no file reads, no re-scoring, no tuning,
# no model execution, no retraining, and no files written.
# =============================================================================

import pandas as pd

MASTER_C5_CONTRACT_628HC5M0 = {
    "reliability_architecture": "uncertainty_only",
    "operating_policy": "uncertainty_only_fixed_threshold",
    "nominal_dev_coverage": 0.8,
    "frozen_exact_risk_threshold": 0.8003345892330226,
    "canonical_test_risk_source": "FD002_TEST_SELECTIVE_RISK_628FCR2",
    "canonical_test_prediction_source": "test_prediction_628hc0r2",
    "prediction_semantic_role": "deterministic endpoint RUL point prediction",
    "test_endpoints": 259,
    "accepted_test_endpoints": 211,
    "abstained_test_endpoints": 48,
    "realized_test_coverage": 0.8146718146718147,
    "realized_test_abstention": 0.18532818532818532,
    "threshold_ties": 0,
    "official_test_labels_read": True,
    "evaluation_target": "piecewise-capped RUL",
    "rul_cap": 125.0,
    "raw_labels_above_cap": 57,
}

MASTER_C5_RISK_STATS_628HC5M0 = pd.DataFrame([{
    "shape": "(259,)",
    "min": 0.0052145304826875915,
    "median": 0.4715610163141138,
    "mean": 0.5026945192062149,
    "max": 0.9950172540970222,
    "sha256":
        "fbd40b69ab28807738d64b374481ac4e71482cd6bede0909087af5aa729ed3ae",
}])

MASTER_C5_PREDICTION_STATS_628HC5M0 = pd.DataFrame([{
    "shape": "(259,)",
    "min": 3.006167411804199,
    "median": 78.24636840820312,
    "mean": 71.0008512804407,
    "max": 126.28646850585938,
    "sha256":
        "10470072f7f027e86d5ec5dfb4a2c8077c13ad914e042e4a7721f066edf37d0b",
}])

MASTER_C5_TEST_PERFORMANCE_628HC5M0 = pd.DataFrame([
    {
        "population": "all",
        "n": 259,
        "mae": 10.618375031644312,
        "rmse": 14.866424796219134,
        "nasa_score_total": 983.282094,
        "nasa_score_mean_displayed": 3.796456,
    },
    {
        "population": "accepted",
        "n": 211,
        "mae": 9.140482108174908,
        "rmse": 13.302282470733848,
        "nasa_score_total": 663.604490,
        "nasa_score_mean_displayed": 3.145045,
    },
    {
        "population": "abstained",
        "n": 48,
        "mae": 17.114946007728577,
        "rmse": 20.363949015827583,
        "nasa_score_total": 319.677604,
        "nasa_score_mean_displayed": 6.659950,
    },
])

MASTER_C5_SELECTIVE_EFFECT_628HC5M0 = pd.DataFrame([{
    "mae_reduction": 1.4778929234694047,
    "relative_mae_reduction": 0.1391825885848887,
    "rmse_reduction": 1.5641423254852853,
    "risk_vs_abs_error_spearman": 0.2627469920465492,
}])

MASTER_C5_HASHES_628HC5M0 = pd.DataFrame([
    {
        "artifact": "canonical_test_risk",
        "sha256":
            "fbd40b69ab28807738d64b374481ac4e71482cd6bede0909087af5aa729ed3ae",
    },
    {
        "artifact": "canonical_test_prediction",
        "sha256":
            "10470072f7f027e86d5ec5dfb4a2c8077c13ad914e042e4a7721f066edf37d0b",
    },
    {
        "artifact": "accept_mask",
        "sha256":
            "8aa52655b56ab24c09941eb0b545b8a7dceabaea10619b7999b247ff885520be",
    },
    {
        "artifact": "abstain_mask",
        "sha256":
            "ecf4e3738b4732d25477e2d67a5517859bda83a602ece8983fd5526388b17559",
    },
    {
        "artifact": "pre_scoring_contract",
        "sha256":
            "291b1bdfbea2f753133868573ba27916653e1de4d4bf1906b4ae86689fe3ec16",
    },
    {
        "artifact": "official_scoring_npz_file",
        "sha256":
            "cc2892a09763c1465ec4b8a19554d7d55ace5cebdc01b09aa24374a6eb12a074",
    },
    {
        "artifact": "official_scoring_csv_file",
        "sha256":
            "0dd4d174c2bba2a8e54297b9eb481ed9c6abb57c9caee980b12bb266983741f3",
    },
    {
        "artifact": "official_score_summary_csv_file",
        "sha256":
            "174a3b2c06b3dce18313b49d545cccb665ff48118159661eced3b573c0f540bd",
    },
    {
        "artifact": "official_scoring_metadata_file",
        "sha256":
            "cf58594183b5ebbabb5fddf77323c8500e5de21a750102e5f13dacc246803a5a",
    },
])

MASTER_C5_STAGE_LEDGER_628HC5M0 = pd.DataFrame([
    {
        "stage": "C5-A0",
        "purpose": "Untouched TEST inference-readiness inventory",
        "status": "PASS",
    },
    {
        "stage": "C5-A0-R1-R1",
        "purpose": "Pre-existing TEST artifact lineage check",
        "status": "PASS",
    },
    {
        "stage": "C5-A1",
        "purpose": "Existing TEST uncertainty-risk lineage reconciliation",
        "status": "PASS",
    },
    {
        "stage": "C5-A2-R2",
        "purpose": "Final canonical TEST reliability-risk identity lock",
        "status": "PASS",
    },
    {
        "stage": "C5-A3",
        "purpose": "Apply frozen abstention policy to canonical TEST risk",
        "status": "PASS",
    },
    {
        "stage": "C5-F1-B",
        "purpose": "Independent reload of TEST risk + operating decisions",
        "status": "PASS",
    },
    {
        "stage": "C5-A4",
        "purpose": "Pre-scoring TEST prediction lineage audit",
        "status": "PASS",
    },
    {
        "stage": "C5-F2-B-R1",
        "purpose": "Independent reload of canonical TEST prediction",
        "status": "PASS",
    },
    {
        "stage": "C5-A5",
        "purpose": "Final pre-scoring TEST contract lock",
        "status": "PASS",
    },
    {
        "stage": "C5-S1",
        "purpose": "One-time official FD002 TEST scoring",
        "status": "PASS",
    },
    {
        "stage": "C5-F3-B-R1",
        "purpose": "Independent reload of frozen official scoring results",
        "status": "PASS",
    },
])

MASTER_C5_RECONCILIATIONS_628HC5M0 = pd.DataFrame([
    {
        "issue":
            "Reproduced TEST final-risk vector was not bitwise-identical "
            "to the historical canonical vector.",
        "resolution":
            "Accepted only after max abs difference "
            "7.66053886991358e-15, atol=1e-14, identical sorted order, "
            "and zero frozen-threshold decision disagreements.",
    },
    {
        "issue":
            "Prediction CSV was not bitwise-identical after text round-trip.",
        "resolution":
            "Accepted as float64 serialization round-trip; max abs difference "
            "1.4210854715202004e-14 under semantic atol=1e-12. "
            "Binary NPZ remained exact.",
    },
    {
        "issue":
            "Initial scoring reload comparison failed on mixed-type CSV columns.",
        "resolution":
            "Repaired with type-aware semantic comparison; scoring CSV max "
            "numerical difference 1.4210854715202004e-14 and summary CSV "
            "max numerical difference 1.7763568394002505e-15.",
    },
])

MASTER_C5_FINAL_AUDIT_628HC5M0 = pd.DataFrame([
    {
        "check": "decision_count_259",
        "passed": (211 + 48 == 259),
    },
    {
        "check": "realized_coverage_exact_from_counts",
        "passed": (211 / 259 == 0.8146718146718147),
    },
    {
        "check": "realized_abstention_exact_from_counts",
        "passed": (48 / 259 == 0.18532818532818532),
    },
    {
        "check": "mae_reduction_consistent",
        "passed": abs(
            (10.618375031644312 - 9.140482108174908)
            - 1.4778929234694047
        ) < 1e-15,
    },
    {
        "check": "rmse_reduction_consistent",
        "passed": abs(
            (14.866424796219134 - 13.302282470733848)
            - 1.5641423254852853
        ) < 1e-15,
    },
    {
        "check": "threshold_unchanged",
        "passed": True,
    },
    {
        "check": "test_scored_once_in_C5_S1",
        "passed": True,
    },
    {
        "check": "no_rescoring_in_this_summary",
        "passed": True,
    },
    {
        "check": "no_model_execution_in_this_summary",
        "passed": True,
    },
    {
        "check": "no_model_retraining_in_this_summary",
        "passed": True,
    },
    {
        "check": "no_files_written_in_this_summary",
        "passed": True,
    },
])

all_master_checks_628hc5m0 = bool(
    MASTER_C5_FINAL_AUDIT_628HC5M0["passed"].all()
)

print("=" * 100)
print("STEP 6.28H-C5-M0 — MASTER FD002 C5 SUMMARY")
print("=" * 100)

print("\n--- FROZEN CONTRACT ---")
for key, value in MASTER_C5_CONTRACT_628HC5M0.items():
    print(f"{key:34s}: {value}")

print("\n--- CANONICAL TEST RISK ---")
display(MASTER_C5_RISK_STATS_628HC5M0)

print("\n--- CANONICAL TEST POINT PREDICTION ---")
display(MASTER_C5_PREDICTION_STATS_628HC5M0)

print("\n--- OFFICIAL TEST PERFORMANCE ---")
display(MASTER_C5_TEST_PERFORMANCE_628HC5M0)

print("\n--- SELECTIVE EFFECT ---")
display(MASTER_C5_SELECTIVE_EFFECT_628HC5M0)

print("\n--- C5 STAGE LEDGER ---")
display(MASTER_C5_STAGE_LEDGER_628HC5M0)

print("\n--- RESOLVED RECONCILIATION NOTES ---")
display(MASTER_C5_RECONCILIATIONS_628HC5M0)

print("\n--- CRITICAL HASHES ---")
display(MASTER_C5_HASHES_628HC5M0)

print("\n--- MASTER AUDIT ---")
display(MASTER_C5_FINAL_AUDIT_628HC5M0)

print("\n--- FINAL C5 CONCLUSION ---")
print("Canonical reliability architecture : uncertainty_only")
print("Frozen operating policy           : uncertainty_only_fixed_threshold")
print("Frozen threshold                  : 0.8003345892330226")
print("TEST accepted / abstained         : 211 / 48")
print("Realized TEST coverage            : 0.8146718146718147")
print("Full TEST MAE                     : 10.618375031644312")
print("Accepted TEST MAE                 : 9.140482108174908")
print("MAE reduction                     : 1.4778929234694047")
print("Relative MAE reduction            : 13.91825885848887%")
print("Risk vs |error| Spearman          : 0.2627469920465492")
print("All master consistency checks     :", all_master_checks_628hc5m0)
print("Official TEST scoring already done: True")
print("New TEST scoring in this cell     : False")
print("Threshold tuned/modified here     : False")
print("Model executed/retrained here     : False")
print("Files written here                : False")

if not all_master_checks_628hc5m0:
    raise RuntimeError("C5 master-summary consistency audit failed.")

print("\n=== END STEP 6.28H-C5-M0 ===")

In [ ]:
# =============================================================================
# STEP 6.28H-C6-A0 — CROSS-SUBSET TRANSFER + PROPOSAL ALIGNMENT LOCK
# =============================================================================
# Purpose:
#   Lock the project-wide methodology for FD001, FD003 and FD004 after the
#   completed FD002 development + official TEST study.
#
# This step:
#   - reads no dataset files
#   - reads no new TEST labels
#   - executes no model
#   - performs no retraining
#   - tunes no threshold
#   - writes no files
# =============================================================================

import pandas as pd

print("=" * 104)
print("STEP 6.28H-C6-A0 — CROSS-SUBSET TRANSFER + PROPOSAL ALIGNMENT LOCK")
print("=" * 104)

# ---------------------------------------------------------------------------
# 1. Research / project lock
# ---------------------------------------------------------------------------

RESEARCH_PROBLEM_628HC6A0 = (
    "Reliability-Aware Selective Remaining Useful Life Prediction "
    "under Sensor Degradation"
)

PROJECT_END_GOAL_628HC6A0 = (
    "Complete the proposal-level FD001-FD004 benchmark, robustness/ablation, "
    "statistics/error analysis, final implementation freeze, then manuscript."
)

FD002_STATUS_628HC6A0 = "COMPLETE_THROUGH_OFFICIAL_TEST_SCORING"

NEXT_DATASET_628HC6A0 = "FD001"

REMAINING_SUBSETS_628HC6A0 = (
    "FD001",
    "FD003",
    "FD004",
)

print("\n--- RESEARCH LOCK ---")
print("Research problem :", RESEARCH_PROBLEM_628HC6A0)
print("Project end goal :", PROJECT_END_GOAL_628HC6A0)
print("FD002 status     :", FD002_STATUS_628HC6A0)
print("Next dataset     :", NEXT_DATASET_628HC6A0)


# ---------------------------------------------------------------------------
# 2. Cross-subset transfer contract
# ---------------------------------------------------------------------------

TRANSFER_CONTRACT_628HC6A0 = pd.DataFrame(
    [
        {
            "component": "research_problem",
            "rule":
                "Keep the original proposal research problem unchanged.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "validation_discipline",
            "rule":
                "Use engine-aware, leakage-safe TRAIN/DEV separation.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "raw_data",
            "rule":
                "Keep official raw files unchanged; preprocessing is code-generated.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "preprocessing_code",
            "rule":
                "Reuse the same validated transformation logic.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "preprocessing_parameters",
            "rule":
                "Fit subset-specific scaling/reference parameters using TRAIN/FIT only.",
            "transfer_status": "REFIT_TRAIN_ONLY",
        },
        {
            "component": "sensor_retention",
            "rule":
                "Keep the same retention rule; derive any subset-specific list "
                "without TEST labels.",
            "transfer_status": "TRAIN_ONLY_RULE",
        },
        {
            "component": "point_prediction_model",
            "rule":
                "Keep the selected architecture/hyperparameter protocol; "
                "train model weights anew for each subset.",
            "transfer_status": "ARCHITECTURE_FROZEN_WEIGHTS_REFIT",
        },
        {
            "component": "corruption_families",
            "rule":
                "Use the same controlled families: bias, noise, drift, dropout, stuck.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "corruption_severity_protocol",
            "rule":
                "Keep severity semantics fixed; derive any required scale statistics "
                "from TRAIN/FIT only.",
            "transfer_status": "FROZEN_RULE_TRAIN_SCALED",
        },
        {
            "component": "reliability_architecture",
            "rule":
                "Use uncertainty-only reliability; do not reopen adaptive-fusion "
                "selection on the remaining subsets.",
            "transfer_status": "FROZEN_FROM_FD002_DEV",
        },
        {
            "component": "mc_dropout_protocol",
            "rule":
                "Preserve frozen stochastic inference: 50 passes, seed 42, ddof=0.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "risk_calibration",
            "rule":
                "Build subset-specific reliability references from TRAIN/FIT only.",
            "transfer_status": "REFIT_TRAIN_ONLY",
        },
        {
            "component": "diagnostic_coverages",
            "rule":
                "Retain 0.90, 0.80 and 0.70 for comparable selective-risk diagnostics.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "operating_policy",
            "rule":
                "Use uncertainty-only fixed-threshold abstention with "
                "nominal coverage 0.80.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "numerical_risk_threshold",
            "rule":
                "Derive the subset threshold from CLEAN DEV only, then freeze it "
                "before official TEST.",
            "transfer_status": "RECOMPUTE_DEV_ONLY",
        },
        {
            "component": "evaluation_metrics",
            "rule":
                "Use the same RUL, reliability and selective-prediction metrics "
                "for every subset.",
            "transfer_status": "FROZEN_GLOBAL",
        },
        {
            "component": "official_test_labels",
            "rule":
                "Never use for tuning/selection; read only after that subset's "
                "complete pre-scoring contract is frozen.",
            "transfer_status": "FINAL_EVALUATION_ONLY",
        },
        {
            "component": "artifact_freeze",
            "rule":
                "Preserve freeze -> hash -> independent reload -> verification.",
            "transfer_status": "FROZEN_GLOBAL",
        },
    ]
)

print("\n--- CROSS-SUBSET TRANSFER CONTRACT ---")
display(TRANSFER_CONTRACT_628HC6A0)


# ---------------------------------------------------------------------------
# 3. Proposal completion roadmap
# ---------------------------------------------------------------------------

PROPOSAL_ROADMAP_628HC6A0 = pd.DataFrame(
    [
        [
            "FD002",
            "Development + frozen policy + official TEST scoring",
            "COMPLETE",
        ],
        [
            "FD001",
            "First cross-subset transfer / replication benchmark",
            "NEXT",
        ],
        [
            "FD003",
            "Second cross-subset benchmark",
            "PENDING",
        ],
        [
            "FD004",
            "Third cross-subset benchmark",
            "PENDING",
        ],
        [
            "Phase 10 consolidation",
            "Combine FD001-FD004 benchmark evidence",
            "PENDING",
        ],
        [
            "Phase 11",
            "Ablations + unseen-corruption robustness",
            "PENDING",
        ],
        [
            "Phase 12",
            "Statistics + error/failure analysis",
            "PENDING",
        ],
        [
            "Phase 13",
            "Freeze final implementation/results/figures",
            "PENDING",
        ],
        [
            "Phase 14",
            "Write journal manuscript",
            "PENDING",
        ],
        [
            "Phase 15",
            "Journal submission",
            "PENDING",
        ],
        [
            "Phase 16",
            "Reviewer response / revision if required",
            "PENDING",
        ],
    ],
    columns=["stage", "purpose", "status"],
)

print("\n--- PROPOSAL-COMPLETION ROADMAP ---")
display(PROPOSAL_ROADMAP_628HC6A0)


# ---------------------------------------------------------------------------
# 4. C6-A0 audit
# ---------------------------------------------------------------------------

reliability_status_628hc6a0 = (
    TRANSFER_CONTRACT_628HC6A0.loc[
        TRANSFER_CONTRACT_628HC6A0["component"]
        == "reliability_architecture",
        "transfer_status",
    ].iloc[0]
)

AUDIT_628HC6A0 = pd.DataFrame(
    [
        [
            "research_problem_locked",
            RESEARCH_PROBLEM_628HC6A0.startswith(
                "Reliability-Aware Selective"
            ),
        ],
        [
            "fd002_marked_complete",
            FD002_STATUS_628HC6A0
            == "COMPLETE_THROUGH_OFFICIAL_TEST_SCORING",
        ],
        [
            "next_subset_fd001",
            NEXT_DATASET_628HC6A0 == "FD001",
        ],
        [
            "remaining_subset_order_exact",
            REMAINING_SUBSETS_628HC6A0
            == ("FD001", "FD003", "FD004"),
        ],
        [
            "uncertainty_only_architecture_locked",
            reliability_status_628hc6a0
            == "FROZEN_FROM_FD002_DEV",
        ],
        [
            "test_labels_reserved_for_final_evaluation",
            True,
        ],
        [
            "no_dataset_file_read_in_this_step",
            True,
        ],
        [
            "no_test_label_read_in_this_step",
            True,
        ],
        [
            "no_model_execution",
            True,
        ],
        [
            "no_model_retraining",
            True,
        ],
        [
            "no_threshold_tuning",
            True,
        ],
        [
            "no_threshold_modification",
            True,
        ],
        [
            "no_files_written",
            True,
        ],
    ],
    columns=["check", "passed"],
)

print("\n--- C6-A0 AUDIT ---")
display(AUDIT_628HC6A0)

ALL_CHECKS_PASSED_628HC6A0 = bool(
    AUDIT_628HC6A0["passed"].all()
)


# ---------------------------------------------------------------------------
# 5. Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 104)
print("STEP 6.28H-C6-A0 FINAL SUMMARY")
print("=" * 104)

print(
    "Research problem locked             :",
    RESEARCH_PROBLEM_628HC6A0,
)
print(
    "FD002 project status                :",
    FD002_STATUS_628HC6A0,
)
print(
    "Next benchmark subset               :",
    NEXT_DATASET_628HC6A0,
)
print(
    "Remaining subset sequence           :",
    REMAINING_SUBSETS_628HC6A0,
)
print(
    "Reliability architecture transfer   : uncertainty_only"
)
print(
    "Operating-policy transfer           : "
    "fixed threshold, nominal coverage 0.80"
)
print(
    "Numeric threshold on new subsets    : "
    "derive from CLEAN DEV only, then freeze"
)
print(
    "Official TEST use on new subsets    : "
    "final evaluation only"
)
print(
    "All C6-A0 checks passed             :",
    ALL_CHECKS_PASSED_628HC6A0,
)
print(
    "Files written                       : False"
)

if not ALL_CHECKS_PASSED_628HC6A0:
    raise RuntimeError(
        "C6-A0 proposal-alignment lock failed."
    )

print(
    "\nNext required step: "
    "C6-A1 — FD001 transfer-readiness inventory "
    "(TRAIN/DEV first; TEST labels untouched)."
)

print("\n=== END STEP 6.28H-C6-A0 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A1 — FD001 TRANSFER-READINESS INVENTORY
# Purpose:
#   1) Confirm the frozen cross-subset research/transfer contract.
#   2) Locate FD001 TRAIN / TEST-feature / official TEST-label files WITHOUT reading them.
#   3) Inventory reusable notebook helpers/objects by name only.
#   4) Verify that no FD001 label-like payload was already loaded before this step.
#
# IMPORTANT:
#   - No dataset file contents are read here.
#   - Official FD001 TEST labels are NOT read.
#   - No model is executed or retrained.
#   - No threshold is tuned or modified.
#   - No files are written.
# ======================================================================================

from pathlib import Path
import inspect
import numpy as np
import pandas as pd

print("=" * 108)
print("STEP 6.28H-C6-A1 — FD001 TRANSFER-READINESS INVENTORY")
print("=" * 108)

# ------------------------------------------------------------------
# 0. Snapshot notebook state BEFORE creating C6-A1 objects
# ------------------------------------------------------------------
globals_before_628hc6a1 = list(globals().items())

# ------------------------------------------------------------------
# 1. Re-assert research / transfer contract
# ------------------------------------------------------------------
research_problem_628hc6a1 = (
    "Reliability-Aware Selective Remaining Useful Life Prediction under Sensor Degradation"
)

transfer_contract_628hc6a1 = pd.DataFrame(
    [
        ["research_problem",
         "keep original proposal problem unchanged",
         "FROZEN_GLOBAL"],

        ["validation",
         "engine-aware leakage-safe TRAIN/DEV separation",
         "FROZEN_GLOBAL"],

        ["preprocessing_code",
         "reuse validated transformation logic",
         "FROZEN_GLOBAL"],

        ["preprocessing_parameters",
         "fit FD001-specific parameters on TRAIN only",
         "REFIT_TRAIN_ONLY"],

        ["sensor_retention",
         "same retention rule; derive FD001 result from TRAIN only",
         "TRAIN_ONLY_RULE"],

        ["point_model",
         "same selected architecture/hyperparameter logic; refit weights on FD001 TRAIN",
         "ARCH_FROZEN_WEIGHTS_REFIT"],

        ["corruptions",
         "same corruption families and severity semantics",
         "FROZEN_GLOBAL"],

        ["reliability_architecture",
         "uncertainty_only",
         "FROZEN_FROM_FD002_DEV"],

        ["mc_dropout",
         "50 passes; frozen stochastic protocol",
         "FROZEN_GLOBAL"],

        ["risk_calibration",
         "FD001-specific reference from permitted TRAIN/CLEAN-DEV data only",
         "REFIT_TRAIN_ONLY"],

        ["diagnostic_coverages",
         "(0.90, 0.80, 0.70)",
         "FROZEN_GLOBAL"],

        ["operating_policy",
         "uncertainty-only fixed-threshold abstention; nominal coverage 0.80",
         "FROZEN_GLOBAL"],

        ["numeric_threshold",
         "derive from FD001 CLEAN DEV only, then freeze",
         "RECOMPUTE_DEV_ONLY"],

        ["official_test_labels",
         "final evaluation only; never tuning/selection",
         "FINAL_EVALUATION_ONLY"],

        ["artifact_protocol",
         "freeze -> hash -> independent reload -> score",
         "FROZEN_GLOBAL"],
    ],
    columns=["component", "rule", "transfer_status"],
)

print("\n--- RESEARCH / TRANSFER LOCK ---")
print("Research problem               :", research_problem_628hc6a1)
print("FD002 reference status        : COMPLETE THROUGH OFFICIAL TEST SCORING")
print("Current transfer subset       : FD001")
print("Reliability architecture      : uncertainty_only")
print("Operating-policy form         : fixed-threshold abstention")
print("Nominal operating coverage    : 0.80")
print("FD001 numeric threshold       : NOT YET DERIVED")
print("Official FD001 TEST-label use : FINAL EVALUATION ONLY")

display(transfer_contract_628hc6a1)

# ------------------------------------------------------------------
# 2. Resolve project root WITHOUT reading any dataset file
# ------------------------------------------------------------------
cwd_628hc6a1 = Path.cwd().resolve()

root_candidates_628hc6a1 = (
    [cwd_628hc6a1] + list(cwd_628hc6a1.parents)
)

project_root_628hc6a1 = next(
    (
        p for p in root_candidates_628hc6a1
        if (p / "02_dataset").exists()
    ),
    cwd_628hc6a1,
)

print("\n--- PROJECT ROOT ---")
print(project_root_628hc6a1)

# ------------------------------------------------------------------
# 3. FD001 official-file PATH inventory only
# ------------------------------------------------------------------
train_matches_628hc6a1 = sorted(
    project_root_628hc6a1.rglob("train_FD001.txt")
)

test_matches_628hc6a1 = sorted(
    project_root_628hc6a1.rglob("test_FD001.txt")
)

rul_matches_628hc6a1 = sorted(
    project_root_628hc6a1.rglob("RUL_FD001.txt")
)

file_inventory_628hc6a1 = pd.DataFrame(
    [
        [
            "TRAIN features",
            "train_FD001.txt",
            len(train_matches_628hc6a1),
            str(train_matches_628hc6a1[0])
            if len(train_matches_628hc6a1) == 1
            else "",
        ],
        [
            "TEST features",
            "test_FD001.txt",
            len(test_matches_628hc6a1),
            str(test_matches_628hc6a1[0])
            if len(test_matches_628hc6a1) == 1
            else "",
        ],
        [
            "TEST labels — path only",
            "RUL_FD001.txt",
            len(rul_matches_628hc6a1),
            str(rul_matches_628hc6a1[0])
            if len(rul_matches_628hc6a1) == 1
            else "",
        ],
    ],
    columns=[
        "role",
        "filename",
        "matches",
        "resolved_path_if_unique",
    ],
)

print("\n--- FD001 FILE-PATH INVENTORY (NO FILE CONTENT READ) ---")
display(file_inventory_628hc6a1)

if len(train_matches_628hc6a1) != 1:
    print(
        "TRAIN path candidates:",
        [str(p) for p in train_matches_628hc6a1]
    )

if len(test_matches_628hc6a1) != 1:
    print(
        "TEST-feature path candidates:",
        [str(p) for p in test_matches_628hc6a1]
    )

if len(rul_matches_628hc6a1) != 1:
    print(
        "TEST-label path candidates:",
        [str(p) for p in rul_matches_628hc6a1]
    )

# ------------------------------------------------------------------
# 4. PRE-EXISTING FD001 label-like payload check
# ------------------------------------------------------------------
label_tokens_628hc6a1 = (
    "LABEL",
    "TARGET",
    "RUL",
)

arraylike_type_names_628hc6a1 = {
    "ndarray",
    "Series",
    "DataFrame",
    "list",
    "tuple",
}

preexisting_fd001_label_payloads_628hc6a1 = []

for name_628hc6a1, obj_628hc6a1 in globals_before_628hc6a1:

    upper_628hc6a1 = str(name_628hc6a1).upper()

    if "FD001" not in upper_628hc6a1:
        continue

    if not any(
        token_628hc6a1 in upper_628hc6a1
        for token_628hc6a1 in label_tokens_628hc6a1
    ):
        continue

    if (
        type(obj_628hc6a1).__name__
        not in arraylike_type_names_628hc6a1
    ):
        continue

    preexisting_fd001_label_payloads_628hc6a1.append(
        {
            "name": str(name_628hc6a1),
            "type": type(obj_628hc6a1).__name__,
        }
    )

preexisting_label_df_628hc6a1 = pd.DataFrame(
    preexisting_fd001_label_payloads_628hc6a1,
    columns=["name", "type"],
)

print("\n--- PRE-EXISTING FD001 LABEL-LIKE PAYLOAD OBJECTS ---")

if len(preexisting_label_df_628hc6a1) == 0:
    print("None found.")
else:
    display(preexisting_label_df_628hc6a1)

# ------------------------------------------------------------------
# 5. Inventory reusable helper functions
#    NAME/SIGNATURE only — nothing is executed.
# ------------------------------------------------------------------
helper_tokens_628hc6a1 = (
    "window",
    "normaliz",
    "predict",
    "dropout",
    "mc_",
    "sensor",
    "corrupt",
    "regime",
)

helper_rows_628hc6a1 = []

for name_628hc6a1, obj_628hc6a1 in globals_before_628hc6a1:

    lower_628hc6a1 = str(name_628hc6a1).lower()

    if not callable(obj_628hc6a1):
        continue

    if not any(
        token_628hc6a1 in lower_628hc6a1
        for token_628hc6a1 in helper_tokens_628hc6a1
    ):
        continue

    try:
        signature_628hc6a1 = str(
            inspect.signature(obj_628hc6a1)
        )
    except Exception:
        signature_628hc6a1 = "<signature unavailable>"

    helper_rows_628hc6a1.append(
        {
            "name": str(name_628hc6a1),
            "type": type(obj_628hc6a1).__name__,
            "signature": signature_628hc6a1,
        }
    )

helpers_df_628hc6a1 = pd.DataFrame(
    helper_rows_628hc6a1,
    columns=["name", "type", "signature"],
).sort_values(
    "name",
    ignore_index=True,
)

print("\n--- REUSABLE PIPELINE HELPERS PRESENT IN NOTEBOOK ---")

if len(helpers_df_628hc6a1) == 0:
    print("No matching helper functions found.")
else:
    display(helpers_df_628hc6a1)

# ------------------------------------------------------------------
# 6. Inventory likely model/scaler/reference objects by NAME only
# ------------------------------------------------------------------
object_tokens_628hc6a1 = (
    "model",
    "gru",
    "scaler",
    "kmeans",
    "reference",
)

object_rows_628hc6a1 = []

for name_628hc6a1, obj_628hc6a1 in globals_before_628hc6a1:

    lower_628hc6a1 = str(name_628hc6a1).lower()

    if callable(obj_628hc6a1):
        continue

    if not any(
        token_628hc6a1 in lower_628hc6a1
        for token_628hc6a1 in object_tokens_628hc6a1
    ):
        continue

    object_rows_628hc6a1.append(
        {
            "name": str(name_628hc6a1),
            "type": type(obj_628hc6a1).__name__,
        }
    )

objects_df_628hc6a1 = pd.DataFrame(
    object_rows_628hc6a1,
    columns=["name", "type"],
).sort_values(
    "name",
    ignore_index=True,
)

print("\n--- RELEVANT EXISTING OBJECT INVENTORY (NAME/TYPE ONLY) ---")

if len(objects_df_628hc6a1) == 0:
    print("No matching objects found.")
else:
    display(objects_df_628hc6a1)

# ------------------------------------------------------------------
# 7. C6-A1 audit
# ------------------------------------------------------------------
unique_train_path_628hc6a1 = (
    len(train_matches_628hc6a1) == 1
)

unique_test_path_628hc6a1 = (
    len(test_matches_628hc6a1) == 1
)

unique_rul_path_628hc6a1 = (
    len(rul_matches_628hc6a1) == 1
)

paths_distinct_628hc6a1 = bool(
    unique_train_path_628hc6a1
    and unique_test_path_628hc6a1
    and unique_rul_path_628hc6a1
    and len(
        {
            train_matches_628hc6a1[0].resolve(),
            test_matches_628hc6a1[0].resolve(),
            rul_matches_628hc6a1[0].resolve(),
        }
    ) == 3
)

audit_628hc6a1 = pd.DataFrame(
    [
        [
            "research_problem_locked",
            research_problem_628hc6a1
            ==
            "Reliability-Aware Selective Remaining Useful Life Prediction under Sensor Degradation",
        ],

        ["fd002_reference_marked_complete", True],

        ["next_subset_fd001", True],

        [
            "reliability_architecture_uncertainty_only",
            True,
        ],

        [
            "operating_policy_form_fixed_threshold",
            True,
        ],

        ["nominal_coverage_080", True],

        [
            "fd001_numeric_threshold_not_yet_derived",
            True,
        ],

        [
            "train_FD001_path_unique",
            unique_train_path_628hc6a1,
        ],

        [
            "test_FD001_path_unique",
            unique_test_path_628hc6a1,
        ],

        [
            "RUL_FD001_path_unique_path_only",
            unique_rul_path_628hc6a1,
        ],

        [
            "three_fd001_paths_distinct",
            paths_distinct_628hc6a1,
        ],

        [
            "no_preexisting_fd001_label_payloads",
            len(
                preexisting_fd001_label_payloads_628hc6a1
            ) == 0,
        ],

        ["no_dataset_file_read_in_this_step", True],

        ["no_test_label_read_in_this_step", True],

        ["no_model_execution", True],

        ["no_model_retraining", True],

        ["no_threshold_tuning", True],

        ["no_threshold_modification", True],

        ["no_files_written", True],
    ],
    columns=["check", "passed"],
)

print("\n--- C6-A1 AUDIT ---")
display(audit_628hc6a1)

all_checks_passed_628hc6a1 = bool(
    audit_628hc6a1["passed"].all()
)

# ------------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 108)
print("STEP 6.28H-C6-A1 FINAL SUMMARY")
print("=" * 108)

print(
    "Research problem locked                 :",
    True,
)

print(
    "FD002 reference benchmark complete      :",
    True,
)

print(
    "Current transfer benchmark              : FD001"
)

print(
    "Unique train_FD001.txt path             :",
    unique_train_path_628hc6a1,
)

print(
    "Unique test_FD001.txt path              :",
    unique_test_path_628hc6a1,
)

print(
    "Unique RUL_FD001.txt path (path only)   :",
    unique_rul_path_628hc6a1,
)

print(
    "Pre-existing FD001 label payloads found :",
    len(preexisting_fd001_label_payloads_628hc6a1),
)

print(
    "Reusable helper candidates found        :",
    len(helpers_df_628hc6a1),
)

print(
    "Relevant existing objects found         :",
    len(objects_df_628hc6a1),
)

print(
    "All C6-A1 checks passed                 :",
    all_checks_passed_628hc6a1,
)

print(
    "Dataset contents read                   :",
    False,
)

print(
    "Official FD001 TEST labels read         :",
    False,
)

print(
    "Model executed/retrained                :",
    False,
)

print(
    "Threshold tuned/modified                :",
    False,
)

print(
    "Files written                           :",
    False,
)

if all_checks_passed_628hc6a1:

    print(
        "\nNext required step: C6-A2 — read FD001 TRAIN only, "
        "validate structure, and establish the engine-aware "
        "TRAIN/DEV split contract."
    )

else:

    print(
        "\nC6-A1 did NOT fully pass. "
        "Do not proceed to FD001 data ingestion yet."
    )

print("\n=== END STEP 6.28H-C6-A1 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A2 — FD001 TRAIN-ONLY INGESTION + ENGINE-AWARE TRAIN/DEV SPLIT CONTRACT
#
# Purpose:
#   1) Read ONLY official train_FD001.txt.
#   2) Validate canonical FD001 TRAIN structure/integrity.
#   3) Establish a deterministic, leakage-safe engine-level FIT/DEV split.
#   4) Freeze the split contract in memory for the next FD001 stages.
#
# IMPORTANT:
#   - test_FD001.txt is NOT read.
#   - RUL_FD001.txt is NOT read.
#   - No model is executed or trained.
#   - No preprocessing parameters are fitted yet.
#   - No reliability threshold is derived/tuned.
#   - No files are written.
# ======================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 112)
print("STEP 6.28H-C6-A2 — FD001 TRAIN-ONLY INGESTION + ENGINE-AWARE TRAIN/DEV SPLIT CONTRACT")
print("=" * 112)

# ------------------------------------------------------------------------------
# 0. Preconditions from C6-A1
# ------------------------------------------------------------------------------
if "all_checks_passed_628hc6a1" not in globals() or not bool(all_checks_passed_628hc6a1):
    raise RuntimeError("C6-A1 must pass before C6-A2.")

if "train_matches_628hc6a1" not in globals() or len(train_matches_628hc6a1) != 1:
    raise RuntimeError("Unique train_FD001.txt path from C6-A1 is required.")

train_path_628hc6a2 = Path(train_matches_628hc6a1[0]).resolve()

if not train_path_628hc6a2.is_file():
    raise FileNotFoundError(f"FD001 TRAIN file not found: {train_path_628hc6a2}")

# ------------------------------------------------------------------------------
# 1. Frozen split protocol
# ------------------------------------------------------------------------------
SPLIT_SEED_628HC6A2 = 42
DEV_ENGINE_FRACTION_628HC6A2 = 0.20

print("\n--- FROZEN VALIDATION DISCIPLINE ---")
print("Dataset                    : FD001")
print("Source used in this step   : train_FD001.txt ONLY")
print("Split unit                 : whole engines")
print("FIT / DEV engine fraction : 0.80 / 0.20")
print("Split seed                 :", SPLIT_SEED_628HC6A2)
print("TEST features read         : False")
print("Official TEST labels read  : False")

# ------------------------------------------------------------------------------
# 2. Read ONLY train_FD001.txt
# ------------------------------------------------------------------------------
column_names_628hc6a2 = (
    ["unit", "cycle", "setting_1", "setting_2", "setting_3"]
    + [f"s{i}" for i in range(1, 22)]
)

FD001_TRAIN_RAW_628HC6A2 = pd.read_csv(
    train_path_628hc6a2,
    sep=r"\s+",
    header=None,
    names=column_names_628hc6a2,
    engine="python",
)

print("\n--- FD001 TRAIN RAW STRUCTURE ---")
print("Path             :", train_path_628hc6a2)
print("Shape            :", FD001_TRAIN_RAW_628HC6A2.shape)
print("Columns          :", len(FD001_TRAIN_RAW_628HC6A2.columns))
print("Unique engines   :", FD001_TRAIN_RAW_628HC6A2["unit"].nunique())
print(
    "Cycle min / max  :",
    int(FD001_TRAIN_RAW_628HC6A2["cycle"].min()),
    "/",
    int(FD001_TRAIN_RAW_628HC6A2["cycle"].max()),
)

# ------------------------------------------------------------------------------
# 3. Structural / numerical integrity checks
# ------------------------------------------------------------------------------
raw_numeric_628hc6a2 = FD001_TRAIN_RAW_628HC6A2.to_numpy(dtype=np.float64)

shape_exact_628hc6a2 = FD001_TRAIN_RAW_628HC6A2.shape == (20631, 26)

column_count_exact_628hc6a2 = (
    len(FD001_TRAIN_RAW_628HC6A2.columns) == 26
)

finite_all_628hc6a2 = bool(
    np.isfinite(raw_numeric_628hc6a2).all()
)

no_missing_628hc6a2 = bool(
    not FD001_TRAIN_RAW_628HC6A2.isna().any().any()
)

unit_values_628hc6a2 = (
    FD001_TRAIN_RAW_628HC6A2["unit"]
    .to_numpy(dtype=np.float64)
)

cycle_values_628hc6a2 = (
    FD001_TRAIN_RAW_628HC6A2["cycle"]
    .to_numpy(dtype=np.float64)
)

unit_integer_like_628hc6a2 = bool(
    np.allclose(
        unit_values_628hc6a2,
        np.round(unit_values_628hc6a2),
        rtol=0.0,
        atol=0.0,
    )
)

cycle_integer_like_628hc6a2 = bool(
    np.allclose(
        cycle_values_628hc6a2,
        np.round(cycle_values_628hc6a2),
        rtol=0.0,
        atol=0.0,
    )
)

engine_ids_628hc6a2 = np.sort(
    FD001_TRAIN_RAW_628HC6A2["unit"]
    .astype(int)
    .unique()
)

engine_count_100_628hc6a2 = (
    len(engine_ids_628hc6a2) == 100
)

engine_ids_1_to_100_628hc6a2 = bool(
    np.array_equal(
        engine_ids_628hc6a2,
        np.arange(1, 101, dtype=int),
    )
)

no_duplicate_unit_cycle_628hc6a2 = bool(
    not FD001_TRAIN_RAW_628HC6A2
    .duplicated(["unit", "cycle"])
    .any()
)

cycle_positive_628hc6a2 = bool(
    (FD001_TRAIN_RAW_628HC6A2["cycle"] >= 1).all()
)

cycle_sequence_rows_628hc6a2 = []

all_cycles_contiguous_628hc6a2 = True
all_cycles_start_at_one_628hc6a2 = True

for engine_id_628hc6a2, group_628hc6a2 in (
    FD001_TRAIN_RAW_628HC6A2.groupby("unit", sort=True)
):

    cycles_628hc6a2 = (
        group_628hc6a2["cycle"]
        .astype(int)
        .to_numpy()
    )

    expected_cycles_628hc6a2 = np.arange(
        1,
        len(cycles_628hc6a2) + 1,
        dtype=int,
    )

    starts_at_one_628hc6a2 = bool(
        len(cycles_628hc6a2) > 0
        and cycles_628hc6a2[0] == 1
    )

    contiguous_628hc6a2 = bool(
        np.array_equal(
            cycles_628hc6a2,
            expected_cycles_628hc6a2,
        )
    )

    all_cycles_start_at_one_628hc6a2 &= (
        starts_at_one_628hc6a2
    )

    all_cycles_contiguous_628hc6a2 &= (
        contiguous_628hc6a2
    )

    cycle_sequence_rows_628hc6a2.append(
        {
            "unit": int(engine_id_628hc6a2),
            "n_cycles": int(len(cycles_628hc6a2)),
            "first_cycle": int(cycles_628hc6a2[0]),
            "last_cycle": int(cycles_628hc6a2[-1]),
            "starts_at_1": starts_at_one_628hc6a2,
            "contiguous": contiguous_628hc6a2,
        }
    )

FD001_ENGINE_CYCLE_AUDIT_628HC6A2 = pd.DataFrame(
    cycle_sequence_rows_628hc6a2
)

# ------------------------------------------------------------------------------
# 4. Deterministic whole-engine FIT/DEV split
# ------------------------------------------------------------------------------
rng_628hc6a2 = np.random.default_rng(
    SPLIT_SEED_628HC6A2
)

permuted_engine_ids_628hc6a2 = (
    rng_628hc6a2.permutation(
        engine_ids_628hc6a2
    )
)

n_dev_engines_628hc6a2 = int(
    round(
        len(engine_ids_628hc6a2)
        * DEV_ENGINE_FRACTION_628HC6A2
    )
)

FD001_DEV_ENGINE_IDS_628HC6A2 = np.sort(
    permuted_engine_ids_628hc6a2[
        :n_dev_engines_628hc6a2
    ]
).astype(int)

FD001_FIT_ENGINE_IDS_628HC6A2 = np.sort(
    permuted_engine_ids_628hc6a2[
        n_dev_engines_628hc6a2:
    ]
).astype(int)

split_disjoint_628hc6a2 = bool(
    np.intersect1d(
        FD001_FIT_ENGINE_IDS_628HC6A2,
        FD001_DEV_ENGINE_IDS_628HC6A2,
    ).size == 0
)

split_union_exact_628hc6a2 = bool(
    np.array_equal(
        np.sort(
            np.concatenate(
                [
                    FD001_FIT_ENGINE_IDS_628HC6A2,
                    FD001_DEV_ENGINE_IDS_628HC6A2,
                ]
            )
        ),
        engine_ids_628hc6a2,
    )
)

split_counts_exact_628hc6a2 = bool(
    len(FD001_FIT_ENGINE_IDS_628HC6A2) == 80
    and len(FD001_DEV_ENGINE_IDS_628HC6A2) == 20
)

FD001_FIT_RAW_628HC6A2 = (
    FD001_TRAIN_RAW_628HC6A2[
        FD001_TRAIN_RAW_628HC6A2["unit"]
        .astype(int)
        .isin(
            FD001_FIT_ENGINE_IDS_628HC6A2
        )
    ]
    .copy()
    .reset_index(drop=True)
)

FD001_DEV_RAW_628HC6A2 = (
    FD001_TRAIN_RAW_628HC6A2[
        FD001_TRAIN_RAW_628HC6A2["unit"]
        .astype(int)
        .isin(
            FD001_DEV_ENGINE_IDS_628HC6A2
        )
    ]
    .copy()
    .reset_index(drop=True)
)

fit_engine_count_628hc6a2 = (
    FD001_FIT_RAW_628HC6A2["unit"]
    .nunique()
)

dev_engine_count_628hc6a2 = (
    FD001_DEV_RAW_628HC6A2["unit"]
    .nunique()
)

row_conservation_628hc6a2 = bool(
    len(FD001_FIT_RAW_628HC6A2)
    + len(FD001_DEV_RAW_628HC6A2)
    ==
    len(FD001_TRAIN_RAW_628HC6A2)
)

# ------------------------------------------------------------------------------
# 5. Split identity hashes
# ------------------------------------------------------------------------------
def sha256_int_array_628hc6a2(arr):

    canonical_628hc6a2 = np.ascontiguousarray(
        arr,
        dtype=np.int64,
    )

    return hashlib.sha256(
        canonical_628hc6a2.tobytes(
            order="C"
        )
    ).hexdigest()


FD001_FIT_ENGINE_IDS_SHA256_628HC6A2 = (
    sha256_int_array_628hc6a2(
        FD001_FIT_ENGINE_IDS_628HC6A2
    )
)

FD001_DEV_ENGINE_IDS_SHA256_628HC6A2 = (
    sha256_int_array_628hc6a2(
        FD001_DEV_ENGINE_IDS_628HC6A2
    )
)

FD001_SPLIT_CONTRACT_628HC6A2 = {
    "dataset": "FD001",
    "source": "train_FD001.txt",
    "split_unit": "engine",
    "fit_engine_fraction": 0.80,
    "dev_engine_fraction": 0.20,
    "seed": int(SPLIT_SEED_628HC6A2),

    "fit_engine_count":
        int(len(FD001_FIT_ENGINE_IDS_628HC6A2)),

    "dev_engine_count":
        int(len(FD001_DEV_ENGINE_IDS_628HC6A2)),

    "fit_row_count":
        int(len(FD001_FIT_RAW_628HC6A2)),

    "dev_row_count":
        int(len(FD001_DEV_RAW_628HC6A2)),

    "fit_engine_ids_sha256":
        FD001_FIT_ENGINE_IDS_SHA256_628HC6A2,

    "dev_engine_ids_sha256":
        FD001_DEV_ENGINE_IDS_SHA256_628HC6A2,

    "test_features_read": False,
    "official_test_labels_read": False,
}

print("\n--- ENGINE-AWARE FIT/DEV SPLIT ---")

print(
    "FIT engines :",
    len(FD001_FIT_ENGINE_IDS_628HC6A2),
)

print(
    "DEV engines :",
    len(FD001_DEV_ENGINE_IDS_628HC6A2),
)

print(
    "FIT rows    :",
    len(FD001_FIT_RAW_628HC6A2),
)

print(
    "DEV rows    :",
    len(FD001_DEV_RAW_628HC6A2),
)

print(
    "Disjoint    :",
    split_disjoint_628hc6a2,
)

print(
    "Union exact :",
    split_union_exact_628hc6a2,
)

print(
    "FIT IDs SHA-256 :",
    FD001_FIT_ENGINE_IDS_SHA256_628HC6A2,
)

print(
    "DEV IDs SHA-256 :",
    FD001_DEV_ENGINE_IDS_SHA256_628HC6A2,
)

print("\nDEV engine IDs:")
print(FD001_DEV_ENGINE_IDS_628HC6A2)

# ------------------------------------------------------------------------------
# 6. Compact engine-length diagnostics
# ------------------------------------------------------------------------------
engine_length_summary_628hc6a2 = (
    FD001_ENGINE_CYCLE_AUDIT_628HC6A2[
        "n_cycles"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame("n_cycles")
)

print("\n--- ENGINE-LENGTH SUMMARY ---")
display(engine_length_summary_628hc6a2)

# ------------------------------------------------------------------------------
# 7. C6-A2 audit
# ------------------------------------------------------------------------------
audit_628hc6a2 = pd.DataFrame(
    [
        [
            "c6_a1_passed",
            bool(all_checks_passed_628hc6a1),
        ],

        ["train_file_only_read", True],

        [
            "raw_shape_20631x26",
            shape_exact_628hc6a2,
        ],

        [
            "column_count_26",
            column_count_exact_628hc6a2,
        ],

        [
            "all_values_finite",
            finite_all_628hc6a2,
        ],

        [
            "no_missing_values",
            no_missing_628hc6a2,
        ],

        [
            "unit_integer_like",
            unit_integer_like_628hc6a2,
        ],

        [
            "cycle_integer_like",
            cycle_integer_like_628hc6a2,
        ],

        [
            "engine_count_100",
            engine_count_100_628hc6a2,
        ],

        [
            "engine_ids_exact_1_to_100",
            engine_ids_1_to_100_628hc6a2,
        ],

        [
            "no_duplicate_unit_cycle",
            no_duplicate_unit_cycle_628hc6a2,
        ],

        [
            "cycles_positive",
            cycle_positive_628hc6a2,
        ],

        [
            "all_engines_start_cycle_1",
            bool(
                all_cycles_start_at_one_628hc6a2
            ),
        ],

        [
            "all_engine_cycles_contiguous",
            bool(
                all_cycles_contiguous_628hc6a2
            ),
        ],

        [
            "fit_dev_engine_counts_80_20",
            split_counts_exact_628hc6a2,
        ],

        [
            "fit_dev_engines_disjoint",
            split_disjoint_628hc6a2,
        ],

        [
            "fit_dev_union_all_100_engines",
            split_union_exact_628hc6a2,
        ],

        [
            "fit_dev_rows_conserved",
            row_conservation_628hc6a2,
        ],

        [
            "fit_engine_count_80",
            fit_engine_count_628hc6a2 == 80,
        ],

        [
            "dev_engine_count_20",
            dev_engine_count_628hc6a2 == 20,
        ],

        ["test_features_not_read", True],

        [
            "official_test_labels_not_read",
            True,
        ],

        ["no_model_execution", True],

        ["no_model_retraining", True],

        ["no_preprocessing_fit_yet", True],

        ["no_threshold_tuning", True],

        ["no_threshold_modification", True],

        ["no_files_written", True],
    ],
    columns=["check", "passed"],
)

print("\n--- C6-A2 AUDIT ---")
display(audit_628hc6a2)

all_checks_passed_628hc6a2 = bool(
    audit_628hc6a2["passed"].all()
)

# ------------------------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 112)
print("STEP 6.28H-C6-A2 FINAL SUMMARY")
print("=" * 112)

print(
    "FD001 TRAIN raw shape               :",
    FD001_TRAIN_RAW_628HC6A2.shape,
)

print(
    "FD001 TRAIN engines                 :",
    len(engine_ids_628hc6a2),
)

print(
    "FIT / DEV engine counts             :",
    len(FD001_FIT_ENGINE_IDS_628HC6A2),
    "/",
    len(FD001_DEV_ENGINE_IDS_628HC6A2),
)

print(
    "FIT / DEV row counts                :",
    len(FD001_FIT_RAW_628HC6A2),
    "/",
    len(FD001_DEV_RAW_628HC6A2),
)

print(
    "Whole-engine leakage prevented      :",
    split_disjoint_628hc6a2,
)

print(
    "Complete engine population retained :",
    split_union_exact_628hc6a2,
)

print(
    "All structural checks passed        :",
    all_checks_passed_628hc6a2,
)

print(
    "TEST features read                  :",
    False,
)

print(
    "Official FD001 TEST labels read     :",
    False,
)

print(
    "Model executed/retrained            :",
    False,
)

print(
    "Preprocessing fitted                :",
    False,
)

print(
    "Threshold tuned/modified            :",
    False,
)

print(
    "Files written                       :",
    False,
)

if all_checks_passed_628hc6a2:

    print(
        "\nNext required step: C6-A3 — derive FD001 FIT-only "
        "target/preprocessing references and verify the "
        "transferred preprocessing contract before model fitting."
    )

else:

    print(
        "\nC6-A2 did NOT fully pass. "
        "Do not proceed to preprocessing or model fitting."
    )

print("\n=== END STEP 6.28H-C6-A2 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A3 — FD001 FIT-ONLY TARGET + PREPROCESSING REFERENCE CONTRACT
# ======================================================================================
#
# Purpose
#   1) Re-establish the frozen C6-A2 whole-engine FIT/DEV split for FD001.
#   2) Derive TRAIN-derived RUL targets with the frozen cap = 125.
#   3) Build preprocessing/statistical references from FIT engines ONLY.
#   4) Produce sensor/operating-setting diagnostics for the next feature-contract step.
#
# Safety
#   - Reads train_FD001.txt only.
#   - Does NOT read test_FD001.txt.
#   - Does NOT read RUL_FD001.txt.
#   - Does NOT execute or retrain any model.
#   - Does NOT tune or modify any abstention threshold.
#   - Does NOT write any files.
# ======================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A3 — FD001 FIT-ONLY TARGET + PREPROCESSING REFERENCE CONTRACT")
print("=" * 118)

# --------------------------------------------------------------------------------------
# 0. Frozen contract constants from completed C6-A2 / FD002 transfer policy
# --------------------------------------------------------------------------------------
DATASET_628HC6A3 = "FD001"
RUL_CAP_628HC6A3 = 125.0
WINDOW_LENGTH_628HC6A3 = 30

EXPECTED_TRAIN_SHAPE_628HC6A3 = (20631, 26)
EXPECTED_FIT_ROWS_628HC6A3 = 16779
EXPECTED_DEV_ROWS_628HC6A3 = 3852

# Exact DEV-engine set already established successfully in C6-A2.
FD001_DEV_ENGINE_IDS_628HC6A3 = np.array(
    [
        3, 9, 19, 22, 25, 26, 28, 34, 43, 51,
        52, 53, 57, 60, 62, 72, 82, 93, 98, 100
    ],
    dtype=np.int64,
)

FD001_ALL_ENGINE_IDS_628HC6A3 = np.arange(1, 101, dtype=np.int64)

FD001_FIT_ENGINE_IDS_628HC6A3 = np.setdiff1d(
    FD001_ALL_ENGINE_IDS_628HC6A3,
    FD001_DEV_ENGINE_IDS_628HC6A3,
    assume_unique=True,
)

# --------------------------------------------------------------------------------------
# 1. Resolve train_FD001.txt ONLY
# --------------------------------------------------------------------------------------
cwd_628hc6a3 = Path.cwd()
known_root_628hc6a3 = Path(r"C:\RUL_Reliability_Research")

relative_train_628hc6a3 = (
    Path("02_dataset")
    / "raw"
    / "CMAPSSData"
    / "train_FD001.txt"
)

train_candidates_628hc6a3 = []

for root_628hc6a3 in (cwd_628hc6a3, known_root_628hc6a3):
    candidate_628hc6a3 = root_628hc6a3 / relative_train_628hc6a3

    if candidate_628hc6a3.exists():
        resolved_628hc6a3 = candidate_628hc6a3.resolve()

        if resolved_628hc6a3 not in train_candidates_628hc6a3:
            train_candidates_628hc6a3.append(resolved_628hc6a3)

if len(train_candidates_628hc6a3) != 1:
    raise RuntimeError(
        "C6-A3 expected exactly one train_FD001.txt at the known project location, "
        f"but found {len(train_candidates_628hc6a3)} candidates: "
        f"{train_candidates_628hc6a3}"
    )

FD001_TRAIN_PATH_628HC6A3 = train_candidates_628hc6a3[0]

if FD001_TRAIN_PATH_628HC6A3.name.lower() != "train_fd001.txt":
    raise RuntimeError(
        "C6-A3 safety guard failed: resolved file is not train_FD001.txt."
    )

# --------------------------------------------------------------------------------------
# 2. Read and validate FD001 TRAIN structure
# --------------------------------------------------------------------------------------
column_names_628hc6a3 = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

FD001_TRAIN_RAW_628HC6A3 = pd.read_csv(
    FD001_TRAIN_PATH_628HC6A3,
    sep=r"\s+",
    header=None,
    engine="python",
)

if FD001_TRAIN_RAW_628HC6A3.shape != EXPECTED_TRAIN_SHAPE_628HC6A3:
    raise RuntimeError(
        "Unexpected FD001 TRAIN shape. "
        f"Expected {EXPECTED_TRAIN_SHAPE_628HC6A3}, "
        f"got {FD001_TRAIN_RAW_628HC6A3.shape}."
    )

FD001_TRAIN_RAW_628HC6A3.columns = column_names_628hc6a3
FD001_TRAIN_RAW_628HC6A3 = FD001_TRAIN_RAW_628HC6A3.copy()

train_numeric_628hc6a3 = (
    FD001_TRAIN_RAW_628HC6A3
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(train_numeric_628hc6a3).all():
    raise RuntimeError(
        "FD001 TRAIN contains non-finite values."
    )

if FD001_TRAIN_RAW_628HC6A3.isna().any().any():
    raise RuntimeError(
        "FD001 TRAIN contains missing values."
    )

unit_float_628hc6a3 = (
    FD001_TRAIN_RAW_628HC6A3["unit"]
    .to_numpy(dtype=np.float64)
)

cycle_float_628hc6a3 = (
    FD001_TRAIN_RAW_628HC6A3["cycle"]
    .to_numpy(dtype=np.float64)
)

if not np.allclose(
    unit_float_628hc6a3,
    np.round(unit_float_628hc6a3),
    rtol=0.0,
    atol=0.0,
):
    raise RuntimeError(
        "FD001 unit identifiers are not integer-like."
    )

if not np.allclose(
    cycle_float_628hc6a3,
    np.round(cycle_float_628hc6a3),
    rtol=0.0,
    atol=0.0,
):
    raise RuntimeError(
        "FD001 cycle identifiers are not integer-like."
    )

FD001_TRAIN_RAW_628HC6A3["unit"] = (
    np.round(unit_float_628hc6a3)
    .astype(np.int64)
)

FD001_TRAIN_RAW_628HC6A3["cycle"] = (
    np.round(cycle_float_628hc6a3)
    .astype(np.int64)
)

observed_ids_628hc6a3 = np.sort(
    FD001_TRAIN_RAW_628HC6A3[
        "unit"
    ].unique().astype(np.int64)
)

if not np.array_equal(
    observed_ids_628hc6a3,
    FD001_ALL_ENGINE_IDS_628HC6A3,
):
    raise RuntimeError(
        "FD001 TRAIN engine population is not exactly IDs 1..100."
    )

# --------------------------------------------------------------------------------------
# 3. Re-establish frozen C6-A2 whole-engine FIT/DEV split
# --------------------------------------------------------------------------------------
FD001_FIT_RAW_628HC6A3 = (
    FD001_TRAIN_RAW_628HC6A3[
        FD001_TRAIN_RAW_628HC6A3[
            "unit"
        ].isin(FD001_FIT_ENGINE_IDS_628HC6A3)
    ]
    .copy()
    .reset_index(drop=True)
)

FD001_DEV_RAW_628HC6A3 = (
    FD001_TRAIN_RAW_628HC6A3[
        FD001_TRAIN_RAW_628HC6A3[
            "unit"
        ].isin(FD001_DEV_ENGINE_IDS_628HC6A3)
    ]
    .copy()
    .reset_index(drop=True)
)

fit_ids_observed_628hc6a3 = np.sort(
    FD001_FIT_RAW_628HC6A3[
        "unit"
    ].unique().astype(np.int64)
)

dev_ids_observed_628hc6a3 = np.sort(
    FD001_DEV_RAW_628HC6A3[
        "unit"
    ].unique().astype(np.int64)
)

if not np.array_equal(
    fit_ids_observed_628hc6a3,
    FD001_FIT_ENGINE_IDS_628HC6A3,
):
    raise RuntimeError(
        "FIT engine IDs do not reproduce the frozen C6-A2 split."
    )

if not np.array_equal(
    dev_ids_observed_628hc6a3,
    FD001_DEV_ENGINE_IDS_628HC6A3,
):
    raise RuntimeError(
        "DEV engine IDs do not reproduce the frozen C6-A2 split."
    )

fit_set_628hc6a3 = set(
    fit_ids_observed_628hc6a3.tolist()
)

dev_set_628hc6a3 = set(
    dev_ids_observed_628hc6a3.tolist()
)

if fit_set_628hc6a3.intersection(dev_set_628hc6a3):
    raise RuntimeError(
        "FIT/DEV engine leakage detected."
    )

if len(FD001_FIT_RAW_628HC6A3) != EXPECTED_FIT_ROWS_628HC6A3:
    raise RuntimeError(
        f"FIT row count mismatch: "
        f"expected {EXPECTED_FIT_ROWS_628HC6A3}, "
        f"got {len(FD001_FIT_RAW_628HC6A3)}."
    )

if len(FD001_DEV_RAW_628HC6A3) != EXPECTED_DEV_ROWS_628HC6A3:
    raise RuntimeError(
        f"DEV row count mismatch: "
        f"expected {EXPECTED_DEV_ROWS_628HC6A3}, "
        f"got {len(FD001_DEV_RAW_628HC6A3)}."
    )

# Stable split hashes.
fit_ids_hash_628hc6a3 = hashlib.sha256(
    np.ascontiguousarray(
        FD001_FIT_ENGINE_IDS_628HC6A3,
        dtype=np.int64,
    ).tobytes()
).hexdigest()

dev_ids_hash_628hc6a3 = hashlib.sha256(
    np.ascontiguousarray(
        FD001_DEV_ENGINE_IDS_628HC6A3,
        dtype=np.int64,
    ).tobytes()
).hexdigest()

# --------------------------------------------------------------------------------------
# 4. Derive deterministic TRAIN-derived RUL targets
#
# Target semantic transferred from our completed pipeline:
# piecewise-capped RUL with cap = 125.
# --------------------------------------------------------------------------------------
def add_fd001_rul_628hc6a3(
    df_628hc6a3,
    cap_628hc6a3=125.0,
):
    out_628hc6a3 = df_628hc6a3.copy()

    max_cycle_628hc6a3 = (
        out_628hc6a3
        .groupby("unit")["cycle"]
        .transform("max")
    )

    rul_raw_628hc6a3 = (
        max_cycle_628hc6a3.to_numpy(
            dtype=np.float64
        )
        -
        out_628hc6a3[
            "cycle"
        ].to_numpy(
            dtype=np.float64
        )
    )

    rul_capped_628hc6a3 = np.minimum(
        rul_raw_628hc6a3,
        float(cap_628hc6a3),
    )

    out_628hc6a3["rul_raw"] = (
        rul_raw_628hc6a3
    )

    out_628hc6a3["rul_capped_125"] = (
        rul_capped_628hc6a3
    )

    return out_628hc6a3


FD001_FIT_LABELED_628HC6A3 = (
    add_fd001_rul_628hc6a3(
        FD001_FIT_RAW_628HC6A3,
        RUL_CAP_628HC6A3,
    )
)

FD001_DEV_LABELED_628HC6A3 = (
    add_fd001_rul_628hc6a3(
        FD001_DEV_RAW_628HC6A3,
        RUL_CAP_628HC6A3,
    )
)

FD001_FIT_TARGET_628HC6A3 = (
    FD001_FIT_LABELED_628HC6A3[
        "rul_capped_125"
    ].to_numpy(dtype=np.float64)
)

FD001_DEV_TARGET_628HC6A3 = (
    FD001_DEV_LABELED_628HC6A3[
        "rul_capped_125"
    ].to_numpy(dtype=np.float64)
)

# Every complete TRAIN trajectory must terminate at RUL = 0.
fit_endpoint_rul_628hc6a3 = (
    FD001_FIT_LABELED_628HC6A3
    .sort_values(["unit", "cycle"])
    .groupby("unit", sort=True)
    .tail(1)["rul_raw"]
    .to_numpy(dtype=np.float64)
)

dev_endpoint_rul_628hc6a3 = (
    FD001_DEV_LABELED_628HC6A3
    .sort_values(["unit", "cycle"])
    .groupby("unit", sort=True)
    .tail(1)["rul_raw"]
    .to_numpy(dtype=np.float64)
)

# --------------------------------------------------------------------------------------
# 5. FIT-only preprocessing/statistical references
#
# IMPORTANT:
# DEV values are NOT used to calculate these parameters.
#
# Sensor retention is NOT selected in this cell.
# That will be the next explicit contract step.
# --------------------------------------------------------------------------------------
OP_COLS_628HC6A3 = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

SENSOR_COLS_628HC6A3 = [
    f"sensor_{i}"
    for i in range(1, 22)
]

CANDIDATE_FEATURE_COLS_628HC6A3 = (
    OP_COLS_628HC6A3
    + SENSOR_COLS_628HC6A3
)

fit_feature_matrix_628hc6a3 = (
    FD001_FIT_RAW_628HC6A3[
        CANDIDATE_FEATURE_COLS_628HC6A3
    ]
    .to_numpy(dtype=np.float64)
)

fit_feature_mean_628hc6a3 = (
    fit_feature_matrix_628hc6a3
    .mean(axis=0)
)

fit_feature_std_628hc6a3 = (
    fit_feature_matrix_628hc6a3
    .std(axis=0, ddof=0)
)

fit_feature_min_628hc6a3 = (
    fit_feature_matrix_628hc6a3
    .min(axis=0)
)

fit_feature_max_628hc6a3 = (
    fit_feature_matrix_628hc6a3
    .max(axis=0)
)

fit_feature_range_628hc6a3 = (
    fit_feature_max_628hc6a3
    -
    fit_feature_min_628hc6a3
)

# Safe scale is stored only as a reference.
# It does NOT yet imply final feature retention.
fit_feature_safe_scale_628hc6a3 = np.where(
    fit_feature_std_628hc6a3 > 0.0,
    fit_feature_std_628hc6a3,
    1.0,
)

FD001_PREPROCESSING_REFERENCE_628HC6A3 = {
    "dataset":
        DATASET_628HC6A3,

    "source":
        "train_FD001.txt",

    "fit_engine_ids":
        FD001_FIT_ENGINE_IDS_628HC6A3.copy(),

    "dev_engine_ids":
        FD001_DEV_ENGINE_IDS_628HC6A3.copy(),

    "fit_engine_ids_sha256":
        fit_ids_hash_628hc6a3,

    "dev_engine_ids_sha256":
        dev_ids_hash_628hc6a3,

    "fit_rows":
        int(len(FD001_FIT_RAW_628HC6A3)),

    "dev_rows":
        int(len(FD001_DEV_RAW_628HC6A3)),

    "rul_cap":
        float(RUL_CAP_628HC6A3),

    "window_length":
        int(WINDOW_LENGTH_628HC6A3),

    "op_cols":
        tuple(OP_COLS_628HC6A3),

    "sensor_cols":
        tuple(SENSOR_COLS_628HC6A3),

    "candidate_feature_cols":
        tuple(CANDIDATE_FEATURE_COLS_628HC6A3),

    "fit_feature_mean":
        fit_feature_mean_628hc6a3.copy(),

    "fit_feature_std_ddof0":
        fit_feature_std_628hc6a3.copy(),

    "fit_feature_safe_scale":
        fit_feature_safe_scale_628hc6a3.copy(),

    "fit_feature_min":
        fit_feature_min_628hc6a3.copy(),

    "fit_feature_max":
        fit_feature_max_628hc6a3.copy(),

    "fit_feature_range":
        fit_feature_range_628hc6a3.copy(),

    "reference_scope":
        "FIT_ENGINES_ONLY",

    "sensor_retention_frozen_here":
        False,

    "model_fitted_here":
        False,

    "threshold_selected_here":
        False,
}

# --------------------------------------------------------------------------------------
# 6. Operating-setting diagnostics
# --------------------------------------------------------------------------------------
operation_rows_628hc6a3 = []

for col_628hc6a3 in OP_COLS_628HC6A3:

    values_628hc6a3 = (
        FD001_FIT_RAW_628HC6A3[
            col_628hc6a3
        ]
        .to_numpy(dtype=np.float64)
    )

    operation_rows_628hc6a3.append(
        {
            "feature":
                col_628hc6a3,

            "fit_mean":
                float(np.mean(values_628hc6a3)),

            "fit_std_ddof0":
                float(np.std(values_628hc6a3, ddof=0)),

            "fit_min":
                float(np.min(values_628hc6a3)),

            "fit_max":
                float(np.max(values_628hc6a3)),

            "fit_range":
                float(
                    np.max(values_628hc6a3)
                    -
                    np.min(values_628hc6a3)
                ),

            "fit_nunique":
                int(
                    pd.Series(
                        values_628hc6a3
                    ).nunique()
                ),
        }
    )

FD001_OPERATION_SETTING_SUMMARY_628HC6A3 = (
    pd.DataFrame(
        operation_rows_628hc6a3
    )
)

# --------------------------------------------------------------------------------------
# 7. Sensor diagnostics
#
# Diagnostic only.
# We do NOT freeze a sensor subset in C6-A3.
# --------------------------------------------------------------------------------------
sensor_rows_628hc6a3 = []

for sensor_628hc6a3 in SENSOR_COLS_628HC6A3:

    values_628hc6a3 = (
        FD001_FIT_RAW_628HC6A3[
            sensor_628hc6a3
        ]
        .to_numpy(dtype=np.float64)
    )

    std_628hc6a3 = float(
        np.std(
            values_628hc6a3,
            ddof=0,
        )
    )

    min_628hc6a3 = float(
        np.min(values_628hc6a3)
    )

    max_628hc6a3 = float(
        np.max(values_628hc6a3)
    )

    range_628hc6a3 = (
        max_628hc6a3
        -
        min_628hc6a3
    )

    sensor_rows_628hc6a3.append(
        {
            "sensor":
                sensor_628hc6a3,

            "fit_mean":
                float(
                    np.mean(
                        values_628hc6a3
                    )
                ),

            "fit_std_ddof0":
                std_628hc6a3,

            "fit_min":
                min_628hc6a3,

            "fit_max":
                max_628hc6a3,

            "fit_range":
                range_628hc6a3,

            "fit_nunique":
                int(
                    pd.Series(
                        values_628hc6a3
                    ).nunique()
                ),

            "constant_exact":
                bool(
                    range_628hc6a3
                    == 0.0
                ),
        }
    )

FD001_SENSOR_SUMMARY_628HC6A3 = (
    pd.DataFrame(
        sensor_rows_628hc6a3
    )
)

# Diagnostic marker only.
# Not a retained/dropped sensor decision.
FD001_SENSOR_SUMMARY_628HC6A3[
    "near_constant_diagnostic"
] = (
    FD001_SENSOR_SUMMARY_628HC6A3[
        "fit_std_ddof0"
    ]
    <= 1e-12
)

# --------------------------------------------------------------------------------------
# 8. Target summaries
# --------------------------------------------------------------------------------------
FD001_TARGET_SUMMARY_628HC6A3 = pd.DataFrame(
    [
        {
            "population":
                "FIT",

            "rows":
                int(
                    len(
                        FD001_FIT_TARGET_628HC6A3
                    )
                ),

            "engines":
                int(
                    len(
                        FD001_FIT_ENGINE_IDS_628HC6A3
                    )
                ),

            "raw_rul_min":
                float(
                    FD001_FIT_LABELED_628HC6A3[
                        "rul_raw"
                    ].min()
                ),

            "raw_rul_max":
                float(
                    FD001_FIT_LABELED_628HC6A3[
                        "rul_raw"
                    ].max()
                ),

            "capped_rul_min":
                float(
                    FD001_FIT_TARGET_628HC6A3.min()
                ),

            "capped_rul_median":
                float(
                    np.median(
                        FD001_FIT_TARGET_628HC6A3
                    )
                ),

            "capped_rul_mean":
                float(
                    np.mean(
                        FD001_FIT_TARGET_628HC6A3
                    )
                ),

            "capped_rul_max":
                float(
                    FD001_FIT_TARGET_628HC6A3.max()
                ),
        },

        {
            "population":
                "DEV",

            "rows":
                int(
                    len(
                        FD001_DEV_TARGET_628HC6A3
                    )
                ),

            "engines":
                int(
                    len(
                        FD001_DEV_ENGINE_IDS_628HC6A3
                    )
                ),

            "raw_rul_min":
                float(
                    FD001_DEV_LABELED_628HC6A3[
                        "rul_raw"
                    ].min()
                ),

            "raw_rul_max":
                float(
                    FD001_DEV_LABELED_628HC6A3[
                        "rul_raw"
                    ].max()
                ),

            "capped_rul_min":
                float(
                    FD001_DEV_TARGET_628HC6A3.min()
                ),

            "capped_rul_median":
                float(
                    np.median(
                        FD001_DEV_TARGET_628HC6A3
                    )
                ),

            "capped_rul_mean":
                float(
                    np.mean(
                        FD001_DEV_TARGET_628HC6A3
                    )
                ),

            "capped_rul_max":
                float(
                    FD001_DEV_TARGET_628HC6A3.max()
                ),
        },
    ]
)

# --------------------------------------------------------------------------------------
# 9. Audit
# --------------------------------------------------------------------------------------
audit_rows_628hc6a3 = [

    (
        "train_file_only_read",
        True,
    ),

    (
        "train_shape_20631x26",
        FD001_TRAIN_RAW_628HC6A3.shape
        == (20631, 26),
    ),

    (
        "engine_population_1_to_100",
        np.array_equal(
            observed_ids_628hc6a3,
            np.arange(1, 101),
        ),
    ),

    (
        "fit_engine_count_80",
        len(
            fit_ids_observed_628hc6a3
        )
        == 80,
    ),

    (
        "dev_engine_count_20",
        len(
            dev_ids_observed_628hc6a3
        )
        == 20,
    ),

    (
        "fit_dev_engine_disjoint",
        len(
            fit_set_628hc6a3
            &
            dev_set_628hc6a3
        )
        == 0,
    ),

    (
        "fit_dev_union_all_100",
        len(
            fit_set_628hc6a3
            |
            dev_set_628hc6a3
        )
        == 100,
    ),

    (
        "fit_rows_16779",
        len(
            FD001_FIT_RAW_628HC6A3
        )
        == 16779,
    ),

    (
        "dev_rows_3852",
        len(
            FD001_DEV_RAW_628HC6A3
        )
        == 3852,
    ),

    (
        "frozen_dev_ids_exact",
        np.array_equal(
            dev_ids_observed_628hc6a3,
            FD001_DEV_ENGINE_IDS_628HC6A3,
        ),
    ),

    (
        "fit_target_finite",
        bool(
            np.isfinite(
                FD001_FIT_TARGET_628HC6A3
            ).all()
        ),
    ),

    (
        "dev_target_finite",
        bool(
            np.isfinite(
                FD001_DEV_TARGET_628HC6A3
            ).all()
        ),
    ),

    (
        "fit_target_within_0_125",
        bool(
            (
                (
                    FD001_FIT_TARGET_628HC6A3
                    >= 0.0
                )
                &
                (
                    FD001_FIT_TARGET_628HC6A3
                    <= 125.0
                )
            ).all()
        ),
    ),

    (
        "dev_target_within_0_125",
        bool(
            (
                (
                    FD001_DEV_TARGET_628HC6A3
                    >= 0.0
                )
                &
                (
                    FD001_DEV_TARGET_628HC6A3
                    <= 125.0
                )
            ).all()
        ),
    ),

    (
        "fit_engine_endpoint_rul_zero",
        bool(
            np.all(
                fit_endpoint_rul_628hc6a3
                == 0.0
            )
        ),
    ),

    (
        "dev_engine_endpoint_rul_zero",
        bool(
            np.all(
                dev_endpoint_rul_628hc6a3
                == 0.0
            )
        ),
    ),

    (
        "candidate_feature_count_24",
        len(
            CANDIDATE_FEATURE_COLS_628HC6A3
        )
        == 24,
    ),

    (
        "fit_reference_rows_exact",
        FD001_PREPROCESSING_REFERENCE_628HC6A3[
            "fit_rows"
        ]
        == 16779,
    ),

    (
        "dev_not_used_for_preprocessing_reference",
        FD001_PREPROCESSING_REFERENCE_628HC6A3[
            "reference_scope"
        ]
        == "FIT_ENGINES_ONLY",
    ),

    (
        "preprocessing_reference_finite",
        bool(
            np.isfinite(
                FD001_PREPROCESSING_REFERENCE_628HC6A3[
                    "fit_feature_mean"
                ]
            ).all()
            and
            np.isfinite(
                FD001_PREPROCESSING_REFERENCE_628HC6A3[
                    "fit_feature_std_ddof0"
                ]
            ).all()
            and
            np.isfinite(
                FD001_PREPROCESSING_REFERENCE_628HC6A3[
                    "fit_feature_safe_scale"
                ]
            ).all()
        ),
    ),

    (
        "test_features_not_read",
        True,
    ),

    (
        "official_test_labels_not_read",
        True,
    ),

    (
        "sensor_retention_not_frozen_here",
        not FD001_PREPROCESSING_REFERENCE_628HC6A3[
            "sensor_retention_frozen_here"
        ],
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning",
        True,
    ),

    (
        "no_threshold_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

C6_A3_AUDIT_628HC6A3 = pd.DataFrame(
    audit_rows_628hc6a3,
    columns=[
        "check",
        "passed",
    ],
)

C6_A3_PASSED_628HC6A3 = bool(
    C6_A3_AUDIT_628HC6A3[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------
# 10. Display
# --------------------------------------------------------------------------------------
print(
    "\n--- FROZEN C6-A2 SPLIT REPRODUCTION ---"
)

print(
    "Dataset                    :",
    DATASET_628HC6A3,
)

print(
    "TRAIN source               :",
    FD001_TRAIN_PATH_628HC6A3,
)

print(
    "FIT engines / rows         :",
    len(
        FD001_FIT_ENGINE_IDS_628HC6A3
    ),
    "/",
    len(
        FD001_FIT_RAW_628HC6A3
    ),
)

print(
    "DEV engines / rows         :",
    len(
        FD001_DEV_ENGINE_IDS_628HC6A3
    ),
    "/",
    len(
        FD001_DEV_RAW_628HC6A3
    ),
)

print(
    "FIT IDs SHA-256            :",
    fit_ids_hash_628hc6a3,
)

print(
    "DEV IDs SHA-256            :",
    dev_ids_hash_628hc6a3,
)

print(
    "TEST features read         :",
    False,
)

print(
    "Official TEST labels read  :",
    False,
)

print(
    "\n--- TARGET CONTRACT ---"
)

print(
    "Target semantic            : piecewise-capped RUL"
)

print(
    "Frozen RUL cap             :",
    RUL_CAP_628HC6A3,
)

display(
    FD001_TARGET_SUMMARY_628HC6A3
)

print(
    "\n--- FIT-ONLY OPERATING-SETTING DIAGNOSTICS ---"
)

display(
    FD001_OPERATION_SETTING_SUMMARY_628HC6A3
)

print(
    "\n--- FIT-ONLY SENSOR DIAGNOSTICS ---"
)

display(
    FD001_SENSOR_SUMMARY_628HC6A3
)

print(
    "\n--- PREPROCESSING REFERENCE CONTRACT ---"
)

print(
    "Reference scope            :",
    FD001_PREPROCESSING_REFERENCE_628HC6A3[
        "reference_scope"
    ],
)

print(
    "Candidate features         :",
    len(
        CANDIDATE_FEATURE_COLS_628HC6A3
    ),
)

print(
    "Operating-setting columns  :",
    OP_COLS_628HC6A3,
)

print(
    "Sensor columns             :",
    len(
        SENSOR_COLS_628HC6A3
    ),
)

print(
    "Sensor retention frozen    :",
    False,
)

print(
    "Model fitted/executed      :",
    False,
)

print(
    "Threshold tuned/modified   :",
    False,
)

print(
    "\n--- C6-A3 AUDIT ---"
)

display(
    C6_A3_AUDIT_628HC6A3
)

# --------------------------------------------------------------------------------------
# 11. Fail safely if any contract check fails
# --------------------------------------------------------------------------------------
if not C6_A3_PASSED_628HC6A3:

    failed_628hc6a3 = (
        C6_A3_AUDIT_628HC6A3.loc[
            ~C6_A3_AUDIT_628HC6A3[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A3 contract failed. Failed checks: "
        +
        ", ".join(
            failed_628hc6a3
        )
    )

# --------------------------------------------------------------------------------------
# 12. Final summary
# --------------------------------------------------------------------------------------
print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A3 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "C6-A2 split reproduced exactly          :",
    True,
)

print(
    "FIT-only target/reference contract      :",
    True,
)

print(
    "RUL cap                                 :",
    RUL_CAP_628HC6A3,
)

print(
    "FIT / DEV engines                       :",
    f"{len(FD001_FIT_ENGINE_IDS_628HC6A3)} / "
    f"{len(FD001_DEV_ENGINE_IDS_628HC6A3)}",
)

print(
    "FIT / DEV rows                          :",
    f"{len(FD001_FIT_RAW_628HC6A3)} / "
    f"{len(FD001_DEV_RAW_628HC6A3)}",
)

print(
    "Preprocessing reference uses DEV values :",
    False,
)

print(
    "Sensor retention selected               :",
    False,
)

print(
    "TEST features read                      :",
    False,
)

print(
    "Official FD001 TEST labels read         :",
    False,
)

print(
    "Model executed/retrained                :",
    False,
)

print(
    "Threshold tuned/modified                :",
    False,
)

print(
    "Files written                           :",
    False,
)

print(
    "All C6-A3 checks passed                 :",
    C6_A3_PASSED_628HC6A3,
)

print(
    "\nNext required step: C6-A4 — derive and lock the FD001 TRAIN-only "
    "sensor-retention + model-feature contract using these FIT-only diagnostics."
)

print(
    "\n=== END STEP 6.28H-C6-A3 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A3-R1 — FD001 FIT-ONLY TARGET + PREPROCESSING REFERENCE CONTRACT
# ======================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A3-R1 — FD001 FIT-ONLY TARGET + PREPROCESSING REFERENCE CONTRACT")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Frozen constants
# ------------------------------------------------------------------
DATASET_628HC6A3R1 = "FD001"
RUL_CAP_628HC6A3R1 = 125.0
WINDOW_LENGTH_628HC6A3R1 = 30
SPLIT_SEED_628HC6A3R1 = 42

EXPECTED_TRAIN_SHAPE_628HC6A3R1 = (20631, 26)
EXPECTED_FIT_ROWS_628HC6A3R1 = 16779
EXPECTED_DEV_ROWS_628HC6A3R1 = 3852

FD001_TRAIN_PATH_628HC6A3R1 = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\train_FD001.txt"
)

if not FD001_TRAIN_PATH_628HC6A3R1.exists():
    raise FileNotFoundError(
        f"train_FD001.txt not found at: {FD001_TRAIN_PATH_628HC6A3R1}"
    )

# ------------------------------------------------------------------
# 1. Read TRAIN only
# ------------------------------------------------------------------
cols_628hc6a3r1 = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

FD001_TRAIN_RAW_628HC6A3R1 = pd.read_csv(
    FD001_TRAIN_PATH_628HC6A3R1,
    sep=r"\s+",
    header=None,
    names=cols_628hc6a3r1,
    engine="python",
)

if FD001_TRAIN_RAW_628HC6A3R1.shape != EXPECTED_TRAIN_SHAPE_628HC6A3R1:
    raise RuntimeError(
        f"Unexpected TRAIN shape: {FD001_TRAIN_RAW_628HC6A3R1.shape}"
    )

if FD001_TRAIN_RAW_628HC6A3R1.isna().any().any():
    raise RuntimeError("Missing values detected in FD001 TRAIN.")

if not np.isfinite(
    FD001_TRAIN_RAW_628HC6A3R1.to_numpy(dtype=np.float64)
).all():
    raise RuntimeError("Non-finite values detected in FD001 TRAIN.")

FD001_TRAIN_RAW_628HC6A3R1["unit"] = (
    FD001_TRAIN_RAW_628HC6A3R1["unit"].astype(np.int64)
)
FD001_TRAIN_RAW_628HC6A3R1["cycle"] = (
    FD001_TRAIN_RAW_628HC6A3R1["cycle"].astype(np.int64)
)

all_ids_628hc6a3r1 = np.sort(
    FD001_TRAIN_RAW_628HC6A3R1["unit"].unique().astype(np.int64)
)

if not np.array_equal(
    all_ids_628hc6a3r1,
    np.arange(1, 101, dtype=np.int64),
):
    raise RuntimeError(
        "FD001 TRAIN engine IDs are not exactly 1..100."
    )

# ------------------------------------------------------------------
# 2. Reproduce frozen C6-A2 split from seed 42
#
# IMPORTANT:
# DEV = first 20 engine IDs from the seeded permutation.
# This avoids manually copying/transcribing the ID list.
# ------------------------------------------------------------------
rng_628hc6a3r1 = np.random.default_rng(
    SPLIT_SEED_628HC6A3R1
)

permuted_ids_628hc6a3r1 = rng_628hc6a3r1.permutation(
    all_ids_628hc6a3r1
)

FD001_DEV_ENGINE_IDS_628HC6A3R1 = np.sort(
    permuted_ids_628hc6a3r1[:20].astype(np.int64)
)

FD001_FIT_ENGINE_IDS_628HC6A3R1 = np.sort(
    permuted_ids_628hc6a3r1[20:].astype(np.int64)
)

EXPECTED_DEV_IDS_628HC6A3R1 = np.array(
    [
        3, 5, 19, 22, 25,
        26, 28, 34, 43, 51,
        52, 53, 57, 60, 62,
        72, 82, 93, 98, 100,
    ],
    dtype=np.int64,
)

if not np.array_equal(
    FD001_DEV_ENGINE_IDS_628HC6A3R1,
    EXPECTED_DEV_IDS_628HC6A3R1,
):
    raise RuntimeError(
        "Seed-42 DEV engine set does not match "
        "the frozen C6-A2 contract."
    )

FD001_FIT_RAW_628HC6A3R1 = (
    FD001_TRAIN_RAW_628HC6A3R1[
        FD001_TRAIN_RAW_628HC6A3R1["unit"].isin(
            FD001_FIT_ENGINE_IDS_628HC6A3R1
        )
    ]
    .copy()
    .reset_index(drop=True)
)

FD001_DEV_RAW_628HC6A3R1 = (
    FD001_TRAIN_RAW_628HC6A3R1[
        FD001_TRAIN_RAW_628HC6A3R1["unit"].isin(
            FD001_DEV_ENGINE_IDS_628HC6A3R1
        )
    ]
    .copy()
    .reset_index(drop=True)
)

fit_rows_628hc6a3r1 = len(
    FD001_FIT_RAW_628HC6A3R1
)

dev_rows_628hc6a3r1 = len(
    FD001_DEV_RAW_628HC6A3R1
)

if fit_rows_628hc6a3r1 != EXPECTED_FIT_ROWS_628HC6A3R1:
    raise RuntimeError(
        f"FIT row count mismatch: "
        f"expected {EXPECTED_FIT_ROWS_628HC6A3R1}, "
        f"got {fit_rows_628hc6a3r1}."
    )

if dev_rows_628hc6a3r1 != EXPECTED_DEV_ROWS_628HC6A3R1:
    raise RuntimeError(
        f"DEV row count mismatch: "
        f"expected {EXPECTED_DEV_ROWS_628HC6A3R1}, "
        f"got {dev_rows_628hc6a3r1}."
    )

if (
    fit_rows_628hc6a3r1
    + dev_rows_628hc6a3r1
    != 20631
):
    raise RuntimeError(
        "FIT + DEV rows do not reconstruct FD001 TRAIN."
    )

if np.intersect1d(
    FD001_FIT_ENGINE_IDS_628HC6A3R1,
    FD001_DEV_ENGINE_IDS_628HC6A3R1,
).size != 0:
    raise RuntimeError(
        "FIT/DEV whole-engine leakage detected."
    )

# ------------------------------------------------------------------
# 3. TRAIN-derived piecewise-capped RUL target
# ------------------------------------------------------------------
def add_fd001_rul_628hc6a3r1(
    df,
    cap=125.0,
):
    out = df.copy()

    max_cycle = (
        out.groupby("unit")["cycle"]
        .transform("max")
        .to_numpy(dtype=np.float64)
    )

    cycle = out[
        "cycle"
    ].to_numpy(dtype=np.float64)

    rul_raw = max_cycle - cycle

    rul_capped = np.minimum(
        rul_raw,
        float(cap),
    )

    out["rul_raw"] = rul_raw
    out["rul_capped_125"] = rul_capped

    return out


FD001_FIT_LABELED_628HC6A3R1 = (
    add_fd001_rul_628hc6a3r1(
        FD001_FIT_RAW_628HC6A3R1,
        RUL_CAP_628HC6A3R1,
    )
)

FD001_DEV_LABELED_628HC6A3R1 = (
    add_fd001_rul_628hc6a3r1(
        FD001_DEV_RAW_628HC6A3R1,
        RUL_CAP_628HC6A3R1,
    )
)

FD001_FIT_TARGET_628HC6A3R1 = (
    FD001_FIT_LABELED_628HC6A3R1[
        "rul_capped_125"
    ].to_numpy(dtype=np.float64)
)

FD001_DEV_TARGET_628HC6A3R1 = (
    FD001_DEV_LABELED_628HC6A3R1[
        "rul_capped_125"
    ].to_numpy(dtype=np.float64)
)

# ------------------------------------------------------------------
# 4. FIT-only preprocessing references
# ------------------------------------------------------------------
OP_COLS_628HC6A3R1 = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

SENSOR_COLS_628HC6A3R1 = [
    f"sensor_{i}"
    for i in range(1, 22)
]

CANDIDATE_FEATURE_COLS_628HC6A3R1 = (
    OP_COLS_628HC6A3R1
    + SENSOR_COLS_628HC6A3R1
)

fit_matrix_628hc6a3r1 = (
    FD001_FIT_RAW_628HC6A3R1[
        CANDIDATE_FEATURE_COLS_628HC6A3R1
    ]
    .to_numpy(dtype=np.float64)
)

fit_mean_628hc6a3r1 = (
    fit_matrix_628hc6a3r1.mean(axis=0)
)

fit_std_628hc6a3r1 = (
    fit_matrix_628hc6a3r1.std(
        axis=0,
        ddof=0,
    )
)

fit_min_628hc6a3r1 = (
    fit_matrix_628hc6a3r1.min(axis=0)
)

fit_max_628hc6a3r1 = (
    fit_matrix_628hc6a3r1.max(axis=0)
)

fit_range_628hc6a3r1 = (
    fit_max_628hc6a3r1
    -
    fit_min_628hc6a3r1
)

fit_safe_scale_628hc6a3r1 = np.where(
    fit_std_628hc6a3r1 > 0.0,
    fit_std_628hc6a3r1,
    1.0,
)

def sha256_int64_628hc6a3r1(x):

    arr = np.ascontiguousarray(
        np.asarray(
            x,
            dtype=np.int64,
        )
    )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


FD001_PREPROCESSING_REFERENCE_628HC6A3R1 = {

    "dataset":
        "FD001",

    "source":
        "train_FD001.txt",

    "reference_scope":
        "FIT_ENGINES_ONLY",

    "split_seed":
        42,

    "fit_engine_ids":
        FD001_FIT_ENGINE_IDS_628HC6A3R1.copy(),

    "dev_engine_ids":
        FD001_DEV_ENGINE_IDS_628HC6A3R1.copy(),

    "fit_engine_ids_sha256":
        sha256_int64_628hc6a3r1(
            FD001_FIT_ENGINE_IDS_628HC6A3R1
        ),

    "dev_engine_ids_sha256":
        sha256_int64_628hc6a3r1(
            FD001_DEV_ENGINE_IDS_628HC6A3R1
        ),

    "fit_rows":
        fit_rows_628hc6a3r1,

    "dev_rows":
        dev_rows_628hc6a3r1,

    "rul_cap":
        125.0,

    "window_length":
        30,

    "candidate_feature_cols":
        tuple(
            CANDIDATE_FEATURE_COLS_628HC6A3R1
        ),

    "fit_feature_mean":
        fit_mean_628hc6a3r1.copy(),

    "fit_feature_std_ddof0":
        fit_std_628hc6a3r1.copy(),

    "fit_feature_safe_scale":
        fit_safe_scale_628hc6a3r1.copy(),

    "fit_feature_min":
        fit_min_628hc6a3r1.copy(),

    "fit_feature_max":
        fit_max_628hc6a3r1.copy(),

    "fit_feature_range":
        fit_range_628hc6a3r1.copy(),

    "sensor_retention_frozen_here":
        False,

    "model_fitted_here":
        False,

    "threshold_selected_here":
        False,
}

# ------------------------------------------------------------------
# 5. FIT-only operating-setting diagnostics
# ------------------------------------------------------------------
operation_rows_628hc6a3r1 = []

for col in OP_COLS_628HC6A3R1:

    x = (
        FD001_FIT_RAW_628HC6A3R1[
            col
        ]
        .to_numpy(dtype=np.float64)
    )

    operation_rows_628hc6a3r1.append(
        {
            "feature":
                col,

            "fit_mean":
                float(np.mean(x)),

            "fit_std_ddof0":
                float(np.std(x, ddof=0)),

            "fit_min":
                float(np.min(x)),

            "fit_max":
                float(np.max(x)),

            "fit_range":
                float(
                    np.max(x)
                    -
                    np.min(x)
                ),

            "fit_nunique":
                int(
                    pd.Series(x).nunique()
                ),
        }
    )

FD001_OPERATION_SETTING_SUMMARY_628HC6A3R1 = (
    pd.DataFrame(
        operation_rows_628hc6a3r1
    )
)

# ------------------------------------------------------------------
# 6. FIT-only sensor diagnostics
#
# Diagnostic only:
# no sensor subset is frozen here.
# ------------------------------------------------------------------
sensor_rows_628hc6a3r1 = []

for sensor in SENSOR_COLS_628HC6A3R1:

    x = (
        FD001_FIT_RAW_628HC6A3R1[
            sensor
        ]
        .to_numpy(dtype=np.float64)
    )

    x_min = float(np.min(x))
    x_max = float(np.max(x))
    x_std = float(
        np.std(x, ddof=0)
    )

    x_range = (
        x_max
        -
        x_min
    )

    sensor_rows_628hc6a3r1.append(
        {
            "sensor":
                sensor,

            "fit_mean":
                float(np.mean(x)),

            "fit_std_ddof0":
                x_std,

            "fit_min":
                x_min,

            "fit_max":
                x_max,

            "fit_range":
                x_range,

            "fit_nunique":
                int(
                    pd.Series(x).nunique()
                ),

            "constant_exact":
                bool(
                    x_range == 0.0
                ),

            "near_constant_diagnostic":
                bool(
                    x_std <= 1e-12
                ),
        }
    )

FD001_SENSOR_SUMMARY_628HC6A3R1 = (
    pd.DataFrame(
        sensor_rows_628hc6a3r1
    )
)

# ------------------------------------------------------------------
# 7. Target summaries
# ------------------------------------------------------------------
FD001_TARGET_SUMMARY_628HC6A3R1 = pd.DataFrame(
    [
        {
            "population":
                "FIT",

            "rows":
                fit_rows_628hc6a3r1,

            "engines":
                80,

            "raw_rul_min":
                float(
                    FD001_FIT_LABELED_628HC6A3R1[
                        "rul_raw"
                    ].min()
                ),

            "raw_rul_max":
                float(
                    FD001_FIT_LABELED_628HC6A3R1[
                        "rul_raw"
                    ].max()
                ),

            "capped_rul_min":
                float(
                    FD001_FIT_TARGET_628HC6A3R1.min()
                ),

            "capped_rul_median":
                float(
                    np.median(
                        FD001_FIT_TARGET_628HC6A3R1
                    )
                ),

            "capped_rul_mean":
                float(
                    np.mean(
                        FD001_FIT_TARGET_628HC6A3R1
                    )
                ),

            "capped_rul_max":
                float(
                    FD001_FIT_TARGET_628HC6A3R1.max()
                ),
        },

        {
            "population":
                "DEV",

            "rows":
                dev_rows_628hc6a3r1,

            "engines":
                20,

            "raw_rul_min":
                float(
                    FD001_DEV_LABELED_628HC6A3R1[
                        "rul_raw"
                    ].min()
                ),

            "raw_rul_max":
                float(
                    FD001_DEV_LABELED_628HC6A3R1[
                        "rul_raw"
                    ].max()
                ),

            "capped_rul_min":
                float(
                    FD001_DEV_TARGET_628HC6A3R1.min()
                ),

            "capped_rul_median":
                float(
                    np.median(
                        FD001_DEV_TARGET_628HC6A3R1
                    )
                ),

            "capped_rul_mean":
                float(
                    np.mean(
                        FD001_DEV_TARGET_628HC6A3R1
                    )
                ),

            "capped_rul_max":
                float(
                    FD001_DEV_TARGET_628HC6A3R1.max()
                ),
        },
    ]
)

# ------------------------------------------------------------------
# 8. Endpoint-RUL validation
# ------------------------------------------------------------------
fit_last_628hc6a3r1 = (
    FD001_FIT_LABELED_628HC6A3R1
    .sort_values(
        ["unit", "cycle"]
    )
    .groupby(
        "unit",
        sort=True,
    )
    .tail(1)[
        "rul_raw"
    ]
    .to_numpy(dtype=np.float64)
)

dev_last_628hc6a3r1 = (
    FD001_DEV_LABELED_628HC6A3R1
    .sort_values(
        ["unit", "cycle"]
    )
    .groupby(
        "unit",
        sort=True,
    )
    .tail(1)[
        "rul_raw"
    ]
    .to_numpy(dtype=np.float64)
)

# ------------------------------------------------------------------
# 9. Audit
# ------------------------------------------------------------------
audit_rows_628hc6a3r1 = [

    (
        "train_shape_20631x26",
        FD001_TRAIN_RAW_628HC6A3R1.shape
        == (20631, 26),
    ),

    (
        "engine_ids_exact_1_to_100",
        np.array_equal(
            all_ids_628hc6a3r1,
            np.arange(1, 101),
        ),
    ),

    (
        "seed42_dev_ids_exact",
        np.array_equal(
            FD001_DEV_ENGINE_IDS_628HC6A3R1,
            EXPECTED_DEV_IDS_628HC6A3R1,
        ),
    ),

    (
        "fit_engine_count_80",
        len(
            FD001_FIT_ENGINE_IDS_628HC6A3R1
        )
        == 80,
    ),

    (
        "dev_engine_count_20",
        len(
            FD001_DEV_ENGINE_IDS_628HC6A3R1
        )
        == 20,
    ),

    (
        "fit_dev_disjoint",
        np.intersect1d(
            FD001_FIT_ENGINE_IDS_628HC6A3R1,
            FD001_DEV_ENGINE_IDS_628HC6A3R1,
        ).size
        == 0,
    ),

    (
        "fit_rows_16779",
        fit_rows_628hc6a3r1
        == 16779,
    ),

    (
        "dev_rows_3852",
        dev_rows_628hc6a3r1
        == 3852,
    ),

    (
        "fit_dev_rows_sum_20631",
        (
            fit_rows_628hc6a3r1
            +
            dev_rows_628hc6a3r1
        )
        == 20631,
    ),

    (
        "fit_target_finite",
        np.isfinite(
            FD001_FIT_TARGET_628HC6A3R1
        ).all(),
    ),

    (
        "dev_target_finite",
        np.isfinite(
            FD001_DEV_TARGET_628HC6A3R1
        ).all(),
    ),

    (
        "fit_target_within_0_125",
        (
            (
                FD001_FIT_TARGET_628HC6A3R1
                >= 0.0
            )
            &
            (
                FD001_FIT_TARGET_628HC6A3R1
                <= 125.0
            )
        ).all(),
    ),

    (
        "dev_target_within_0_125",
        (
            (
                FD001_DEV_TARGET_628HC6A3R1
                >= 0.0
            )
            &
            (
                FD001_DEV_TARGET_628HC6A3R1
                <= 125.0
            )
        ).all(),
    ),

    (
        "fit_endpoint_rul_zero",
        np.all(
            fit_last_628hc6a3r1
            == 0.0
        ),
    ),

    (
        "dev_endpoint_rul_zero",
        np.all(
            dev_last_628hc6a3r1
            == 0.0
        ),
    ),

    (
        "candidate_feature_count_24",
        len(
            CANDIDATE_FEATURE_COLS_628HC6A3R1
        )
        == 24,
    ),

    (
        "reference_scope_fit_only",
        FD001_PREPROCESSING_REFERENCE_628HC6A3R1[
            "reference_scope"
        ]
        == "FIT_ENGINES_ONLY",
    ),

    (
        "preprocessing_reference_finite",
        (
            np.isfinite(
                fit_mean_628hc6a3r1
            ).all()
            and
            np.isfinite(
                fit_std_628hc6a3r1
            ).all()
            and
            np.isfinite(
                fit_safe_scale_628hc6a3r1
            ).all()
        ),
    ),

    (
        "test_features_not_read",
        True,
    ),

    (
        "official_test_labels_not_read",
        True,
    ),

    (
        "sensor_retention_not_selected_here",
        True,
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning",
        True,
    ),

    (
        "no_threshold_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

C6_A3_AUDIT_628HC6A3R1 = pd.DataFrame(
    audit_rows_628hc6a3r1,
    columns=[
        "check",
        "passed",
    ],
)

C6_A3_PASSED_628HC6A3R1 = bool(
    C6_A3_AUDIT_628HC6A3R1[
        "passed"
    ].all()
)

# ------------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------------
print(
    "\n--- REPRODUCED C6-A2 SPLIT ---"
)

print(
    "DEV engine IDs :",
    FD001_DEV_ENGINE_IDS_628HC6A3R1.tolist(),
)

print(
    "FIT engines    :",
    len(
        FD001_FIT_ENGINE_IDS_628HC6A3R1
    ),
)

print(
    "DEV engines    :",
    len(
        FD001_DEV_ENGINE_IDS_628HC6A3R1
    ),
)

print(
    "FIT rows       :",
    fit_rows_628hc6a3r1,
)

print(
    "DEV rows       :",
    dev_rows_628hc6a3r1,
)

print(
    "\n--- TARGET SUMMARY ---"
)

display(
    FD001_TARGET_SUMMARY_628HC6A3R1
)

print(
    "\n--- FIT-ONLY OPERATING-SETTING DIAGNOSTICS ---"
)

display(
    FD001_OPERATION_SETTING_SUMMARY_628HC6A3R1
)

print(
    "\n--- FIT-ONLY SENSOR DIAGNOSTICS ---"
)

display(
    FD001_SENSOR_SUMMARY_628HC6A3R1
)

print(
    "\n--- C6-A3-R1 AUDIT ---"
)

display(
    C6_A3_AUDIT_628HC6A3R1
)

# ------------------------------------------------------------------
# 11. Fail safely
# ------------------------------------------------------------------
if not C6_A3_PASSED_628HC6A3R1:

    failed = (
        C6_A3_AUDIT_628HC6A3R1.loc[
            ~C6_A3_AUDIT_628HC6A3R1[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A3-R1 contract failed. Failed checks: "
        +
        ", ".join(failed)
    )

# ------------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------------
print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A3-R1 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "C6-A2 split reproduced exactly          :",
    True,
)

print(
    "Correct DEV engine ID includes 5 not 9 :",
    True,
)

print(
    "FIT / DEV engines                       : 80 / 20"
)

print(
    "FIT / DEV rows                          : "
    f"{fit_rows_628hc6a3r1} / "
    f"{dev_rows_628hc6a3r1}"
)

print(
    "RUL cap                                 :",
    RUL_CAP_628HC6A3R1,
)

print(
    "Preprocessing reference scope           : FIT_ENGINES_ONLY"
)

print(
    "Sensor retention selected               :",
    False,
)

print(
    "TEST features read                      :",
    False,
)

print(
    "Official FD001 TEST labels read         :",
    False,
)

print(
    "Model executed/retrained                :",
    False,
)

print(
    "Threshold tuned/modified                :",
    False,
)

print(
    "Files written                           :",
    False,
)

print(
    "All C6-A3-R1 checks passed              :",
    C6_A3_PASSED_628HC6A3R1,
)

print(
    "\nNext required step: C6-A4 — derive and lock "
    "the FD001 TRAIN-only sensor-retention + "
    "model-feature contract."
)

print(
    "\n=== END STEP 6.28H-C6-A3-R1 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A4 — FD001 TRAIN-ONLY SENSOR RETENTION + MODEL-FEATURE CONTRACT
# ======================================================================================

import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A4 — FD001 TRAIN-ONLY SENSOR RETENTION + MODEL-FEATURE CONTRACT")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Require successful C6-A3-R1 state
# ------------------------------------------------------------------
if "C6_A3_PASSED_628HC6A3R1" not in globals():
    raise RuntimeError(
        "C6-A3-R1 state not found. Run the successful C6-A3-R1 cell first."
    )

if not bool(C6_A3_PASSED_628HC6A3R1):
    raise RuntimeError(
        "C6-A3-R1 did not pass. C6-A4 must not proceed."
    )

required_objects_628hc6a4 = [
    "FD001_FIT_RAW_628HC6A3R1",
    "FD001_DEV_RAW_628HC6A3R1",
    "FD001_SENSOR_SUMMARY_628HC6A3R1",
    "FD001_OPERATION_SETTING_SUMMARY_628HC6A3R1",
    "FD001_FIT_ENGINE_IDS_628HC6A3R1",
    "FD001_DEV_ENGINE_IDS_628HC6A3R1",
    "FD001_FIT_TARGET_628HC6A3R1",
    "FD001_DEV_TARGET_628HC6A3R1",
]

missing_628hc6a4 = [
    name
    for name in required_objects_628hc6a4
    if name not in globals()
]

if missing_628hc6a4:
    raise RuntimeError(
        "Missing required C6-A3-R1 objects: "
        + ", ".join(missing_628hc6a4)
    )

# ------------------------------------------------------------------
# 1. Frozen transfer rule
#
# Same rule transferred from FD002:
# retain non-constant FIT-only features;
# drop exact-constant features only.
#
# IMPORTANT:
# This is NOT a low-variance filtering rule.
# sensor_6 therefore remains retained.
# ------------------------------------------------------------------
SENSOR_RETENTION_RULE_628HC6A4 = (
    "retain sensor iff FIT-only max > FIT-only min; "
    "drop exact-constant sensors only"
)

OP_RETENTION_RULE_628HC6A4 = (
    "retain operating setting iff FIT-only max > FIT-only min; "
    "drop exact-constant operating settings only"
)

sensor_summary_628hc6a4 = (
    FD001_SENSOR_SUMMARY_628HC6A3R1
    .copy()
    .reset_index(drop=True)
)

op_summary_628hc6a4 = (
    FD001_OPERATION_SETTING_SUMMARY_628HC6A3R1
    .copy()
    .reset_index(drop=True)
)

FD001_RETAINED_SENSORS_628HC6A4 = (
    sensor_summary_628hc6a4.loc[
        sensor_summary_628hc6a4["fit_range"] > 0.0,
        "sensor",
    ]
    .astype(str)
    .tolist()
)

FD001_DROPPED_SENSORS_628HC6A4 = (
    sensor_summary_628hc6a4.loc[
        sensor_summary_628hc6a4["fit_range"] == 0.0,
        "sensor",
    ]
    .astype(str)
    .tolist()
)

FD001_RETAINED_OP_SETTINGS_628HC6A4 = (
    op_summary_628hc6a4.loc[
        op_summary_628hc6a4["fit_range"] > 0.0,
        "feature",
    ]
    .astype(str)
    .tolist()
)

FD001_DROPPED_OP_SETTINGS_628HC6A4 = (
    op_summary_628hc6a4.loc[
        op_summary_628hc6a4["fit_range"] == 0.0,
        "feature",
    ]
    .astype(str)
    .tolist()
)

FD001_MODEL_FEATURES_628HC6A4 = (
    FD001_RETAINED_OP_SETTINGS_628HC6A4
    + FD001_RETAINED_SENSORS_628HC6A4
)

# ------------------------------------------------------------------
# 2. Expected FD001 result from frozen rule
# ------------------------------------------------------------------
EXPECTED_RETAINED_OPS_628HC6A4 = [
    "op_setting_1",
    "op_setting_2",
]

EXPECTED_DROPPED_OPS_628HC6A4 = [
    "op_setting_3",
]

EXPECTED_DROPPED_SENSORS_628HC6A4 = [
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19",
]

EXPECTED_RETAINED_SENSORS_628HC6A4 = [
    f"sensor_{i}"
    for i in [
        2, 3, 4, 6, 7,
        8, 9, 11, 12, 13,
        14, 15, 17, 20, 21,
    ]
]

EXPECTED_MODEL_FEATURES_628HC6A4 = (
    EXPECTED_RETAINED_OPS_628HC6A4
    + EXPECTED_RETAINED_SENSORS_628HC6A4
)

# ------------------------------------------------------------------
# 3. FIT-only normalization parameters for locked features
# ------------------------------------------------------------------
fit_feature_matrix_628hc6a4 = (
    FD001_FIT_RAW_628HC6A3R1[
        FD001_MODEL_FEATURES_628HC6A4
    ]
    .to_numpy(dtype=np.float64)
)

dev_feature_matrix_628hc6a4 = (
    FD001_DEV_RAW_628HC6A3R1[
        FD001_MODEL_FEATURES_628HC6A4
    ]
    .to_numpy(dtype=np.float64)
)

FD001_MODEL_FEATURE_MEAN_628HC6A4 = (
    fit_feature_matrix_628hc6a4.mean(axis=0)
)

FD001_MODEL_FEATURE_STD_628HC6A4 = (
    fit_feature_matrix_628hc6a4.std(
        axis=0,
        ddof=0,
    )
)

if np.any(FD001_MODEL_FEATURE_STD_628HC6A4 <= 0.0):
    raise RuntimeError(
        "A retained FD001 model feature has zero FIT standard deviation."
    )

FD001_FIT_NORMALIZED_FEATURES_628HC6A4 = (
    (
        fit_feature_matrix_628hc6a4
        - FD001_MODEL_FEATURE_MEAN_628HC6A4
    )
    / FD001_MODEL_FEATURE_STD_628HC6A4
)

FD001_DEV_NORMALIZED_FEATURES_628HC6A4 = (
    (
        dev_feature_matrix_628hc6a4
        - FD001_MODEL_FEATURE_MEAN_628HC6A4
    )
    / FD001_MODEL_FEATURE_STD_628HC6A4
)

# ------------------------------------------------------------------
# 4. Reproducibility hashes
# ------------------------------------------------------------------
def sha256_text_628hc6a4(values):
    payload = json.dumps(
        list(values),
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def sha256_float64_628hc6a4(values):
    arr = np.ascontiguousarray(
        np.asarray(
            values,
            dtype=np.float64,
        )
    )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


FD001_MODEL_FEATURES_SHA256_628HC6A4 = (
    sha256_text_628hc6a4(
        FD001_MODEL_FEATURES_628HC6A4
    )
)

FD001_MODEL_FEATURE_MEAN_SHA256_628HC6A4 = (
    sha256_float64_628hc6a4(
        FD001_MODEL_FEATURE_MEAN_628HC6A4
    )
)

FD001_MODEL_FEATURE_STD_SHA256_628HC6A4 = (
    sha256_float64_628hc6a4(
        FD001_MODEL_FEATURE_STD_628HC6A4
    )
)

# ------------------------------------------------------------------
# 5. Historical FD002 architecture dimensionality cross-check
# ------------------------------------------------------------------
FD002_FEATURE_CONTRACT_PRESENT_628HC6A4 = (
    "FD002_MODEL_FEATURES_628C" in globals()
)

if FD002_FEATURE_CONTRACT_PRESENT_628HC6A4:
    FD002_MODEL_FEATURE_COUNT_628HC6A4 = len(
        list(FD002_MODEL_FEATURES_628C)
    )
else:
    FD002_MODEL_FEATURE_COUNT_628HC6A4 = None

# ------------------------------------------------------------------
# 6. Locked in-memory FD001 feature contract
# ------------------------------------------------------------------
FD001_MODEL_FEATURE_CONTRACT_628HC6A4 = {
    "dataset": "FD001",
    "derivation_scope": "FIT_ENGINES_ONLY",
    "split_seed": 42,
    "window_length": 30,
    "rul_cap": 125.0,
    "sensor_retention_rule": SENSOR_RETENTION_RULE_628HC6A4,
    "op_retention_rule": OP_RETENTION_RULE_628HC6A4,
    "retained_operating_settings": tuple(
        FD001_RETAINED_OP_SETTINGS_628HC6A4
    ),
    "dropped_operating_settings": tuple(
        FD001_DROPPED_OP_SETTINGS_628HC6A4
    ),
    "retained_sensors": tuple(
        FD001_RETAINED_SENSORS_628HC6A4
    ),
    "dropped_sensors": tuple(
        FD001_DROPPED_SENSORS_628HC6A4
    ),
    "model_features": tuple(
        FD001_MODEL_FEATURES_628HC6A4
    ),
    "model_feature_count": len(
        FD001_MODEL_FEATURES_628HC6A4
    ),
    "feature_names_sha256":
        FD001_MODEL_FEATURES_SHA256_628HC6A4,
    "fit_mean_sha256":
        FD001_MODEL_FEATURE_MEAN_SHA256_628HC6A4,
    "fit_std_sha256":
        FD001_MODEL_FEATURE_STD_SHA256_628HC6A4,
    "official_test_features_used": False,
    "official_test_labels_used": False,
    "model_fitted_here": False,
    "threshold_selected_here": False,
}

# ------------------------------------------------------------------
# 7. Human-readable retention table
# ------------------------------------------------------------------
feature_retention_rows_628hc6a4 = []

for feature in [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]:
    row = op_summary_628hc6a4.loc[
        op_summary_628hc6a4["feature"] == feature
    ].iloc[0]

    feature_retention_rows_628hc6a4.append(
        {
            "feature": feature,
            "kind": "operating_setting",
            "fit_std_ddof0": float(
                row["fit_std_ddof0"]
            ),
            "fit_range": float(
                row["fit_range"]
            ),
            "fit_nunique": int(
                row["fit_nunique"]
            ),
            "retained": bool(
                feature
                in FD001_RETAINED_OP_SETTINGS_628HC6A4
            ),
        }
    )

for sensor in [
    f"sensor_{i}"
    for i in range(1, 22)
]:
    row = sensor_summary_628hc6a4.loc[
        sensor_summary_628hc6a4["sensor"] == sensor
    ].iloc[0]

    feature_retention_rows_628hc6a4.append(
        {
            "feature": sensor,
            "kind": "sensor",
            "fit_std_ddof0": float(
                row["fit_std_ddof0"]
            ),
            "fit_range": float(
                row["fit_range"]
            ),
            "fit_nunique": int(
                row["fit_nunique"]
            ),
            "retained": bool(
                sensor
                in FD001_RETAINED_SENSORS_628HC6A4
            ),
        }
    )

FD001_FEATURE_RETENTION_TABLE_628HC6A4 = (
    pd.DataFrame(
        feature_retention_rows_628hc6a4
    )
)

# ------------------------------------------------------------------
# 8. Audit
# ------------------------------------------------------------------
audit_rows_628hc6a4 = [
    (
        "c6_a3_r1_passed",
        bool(C6_A3_PASSED_628HC6A3R1),
    ),
    (
        "retention_derived_from_fit_only",
        True,
    ),
    (
        "retained_ops_exact",
        FD001_RETAINED_OP_SETTINGS_628HC6A4
        == EXPECTED_RETAINED_OPS_628HC6A4,
    ),
    (
        "dropped_ops_exact",
        FD001_DROPPED_OP_SETTINGS_628HC6A4
        == EXPECTED_DROPPED_OPS_628HC6A4,
    ),
    (
        "retained_sensor_count_15",
        len(
            FD001_RETAINED_SENSORS_628HC6A4
        )
        == 15,
    ),
    (
        "dropped_sensor_count_6",
        len(
            FD001_DROPPED_SENSORS_628HC6A4
        )
        == 6,
    ),
    (
        "retained_sensors_exact",
        FD001_RETAINED_SENSORS_628HC6A4
        == EXPECTED_RETAINED_SENSORS_628HC6A4,
    ),
    (
        "dropped_sensors_exact",
        FD001_DROPPED_SENSORS_628HC6A4
        == EXPECTED_DROPPED_SENSORS_628HC6A4,
    ),
    (
        "sensor_6_retained_by_exact_constant_rule",
        "sensor_6"
        in FD001_RETAINED_SENSORS_628HC6A4,
    ),
    (
        "model_feature_count_17",
        len(
            FD001_MODEL_FEATURES_628HC6A4
        )
        == 17,
    ),
    (
        "model_features_exact",
        FD001_MODEL_FEATURES_628HC6A4
        == EXPECTED_MODEL_FEATURES_628HC6A4,
    ),
    (
        "all_retained_feature_std_positive",
        bool(
            np.all(
                FD001_MODEL_FEATURE_STD_628HC6A4
                > 0.0
            )
        ),
    ),
    (
        "fit_normalized_shape_16779x17",
        FD001_FIT_NORMALIZED_FEATURES_628HC6A4.shape
        == (16779, 17),
    ),
    (
        "dev_normalized_shape_3852x17",
        FD001_DEV_NORMALIZED_FEATURES_628HC6A4.shape
        == (3852, 17),
    ),
    (
        "fit_normalized_finite",
        bool(
            np.isfinite(
                FD001_FIT_NORMALIZED_FEATURES_628HC6A4
            ).all()
        ),
    ),
    (
        "dev_normalized_finite",
        bool(
            np.isfinite(
                FD001_DEV_NORMALIZED_FEATURES_628HC6A4
            ).all()
        ),
    ),
    (
        "fit_normalized_mean_near_zero",
        bool(
            np.max(
                np.abs(
                    FD001_FIT_NORMALIZED_FEATURES_628HC6A4.mean(
                        axis=0
                    )
                )
            )
            < 1e-10
        ),
    ),
    (
        "fit_normalized_std_near_one",
        bool(
            np.max(
                np.abs(
                    FD001_FIT_NORMALIZED_FEATURES_628HC6A4.std(
                        axis=0,
                        ddof=0,
                    )
                    - 1.0
                )
            )
            < 1e-10
        ),
    ),
    (
        "fd002_feature_count_17_if_present",
        (
            not FD002_FEATURE_CONTRACT_PRESENT_628HC6A4
            or FD002_MODEL_FEATURE_COUNT_628HC6A4 == 17
        ),
    ),
    (
        "test_features_not_read",
        True,
    ),
    (
        "official_test_labels_not_read",
        True,
    ),
    (
        "no_model_execution",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning",
        True,
    ),
    (
        "no_threshold_modification",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

C6_A4_AUDIT_628HC6A4 = pd.DataFrame(
    audit_rows_628hc6a4,
    columns=[
        "check",
        "passed",
    ],
)

C6_A4_PASSED_628HC6A4 = bool(
    C6_A4_AUDIT_628HC6A4[
        "passed"
    ].all()
)

# ------------------------------------------------------------------
# 9. Display
# ------------------------------------------------------------------
print(
    "\n--- FD001 FIT-ONLY FEATURE RETENTION ---"
)

display(
    FD001_FEATURE_RETENTION_TABLE_628HC6A4
)

print(
    "\n--- LOCKED MODEL FEATURE SET ---"
)

print(
    "Retained operating settings :",
    FD001_RETAINED_OP_SETTINGS_628HC6A4,
)

print(
    "Dropped operating settings  :",
    FD001_DROPPED_OP_SETTINGS_628HC6A4,
)

print(
    "Retained sensors            :",
    FD001_RETAINED_SENSORS_628HC6A4,
)

print(
    "Dropped sensors             :",
    FD001_DROPPED_SENSORS_628HC6A4,
)

print(
    "Model feature count         :",
    len(
        FD001_MODEL_FEATURES_628HC6A4
    ),
)

print(
    "Model features              :",
    FD001_MODEL_FEATURES_628HC6A4,
)

print(
    "Feature-name SHA-256        :",
    FD001_MODEL_FEATURES_SHA256_628HC6A4,
)

if FD002_FEATURE_CONTRACT_PRESENT_628HC6A4:
    print(
        "Historical FD002 feature count:",
        FD002_MODEL_FEATURE_COUNT_628HC6A4,
    )
else:
    print(
        "Historical FD002 feature count: "
        "object not present; not required."
    )

print(
    "\n--- C6-A4 AUDIT ---"
)

display(
    C6_A4_AUDIT_628HC6A4
)

# ------------------------------------------------------------------
# 10. Fail safely
# ------------------------------------------------------------------
if not C6_A4_PASSED_628HC6A4:

    failed_628hc6a4 = (
        C6_A4_AUDIT_628HC6A4.loc[
            ~C6_A4_AUDIT_628HC6A4[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A4 feature-contract lock failed. Failed checks: "
        + ", ".join(
            failed_628hc6a4
        )
    )

# ------------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------------
print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A4 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "Retention rule                    : "
    "exact FIT-only constant-feature removal"
)

print(
    "Retained / dropped op settings    : "
    f"{len(FD001_RETAINED_OP_SETTINGS_628HC6A4)} / "
    f"{len(FD001_DROPPED_OP_SETTINGS_628HC6A4)}"
)

print(
    "Retained / dropped sensors        : "
    f"{len(FD001_RETAINED_SENSORS_628HC6A4)} / "
    f"{len(FD001_DROPPED_SENSORS_628HC6A4)}"
)

print(
    "FD001 model input features        :",
    len(
        FD001_MODEL_FEATURES_628HC6A4
    ),
)

print(
    "sensor_6 retained                 :",
    "sensor_6"
    in FD001_RETAINED_SENSORS_628HC6A4,
)

print(
    "FIT normalized shape              :",
    FD001_FIT_NORMALIZED_FEATURES_628HC6A4.shape,
)

print(
    "DEV normalized shape              :",
    FD001_DEV_NORMALIZED_FEATURES_628HC6A4.shape,
)

print(
    "TEST features read                :",
    False,
)

print(
    "Official FD001 TEST labels read   :",
    False,
)

print(
    "Model executed/retrained          :",
    False,
)

print(
    "Threshold tuned/modified          :",
    False,
)

print(
    "Files written                     :",
    False,
)

print(
    "All C6-A4 checks passed           :",
    C6_A4_PASSED_628HC6A4,
)

print(
    "\nNext required step: C6-A5 — construct and audit "
    "the FD001 30-cycle FIT/DEV window contract using "
    "these 17 locked features and capped-RUL target, "
    "before any model fitting."
)

print(
    "\n=== END STEP 6.28H-C6-A4 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A5-R1 — FD001 30-CYCLE FIT/DEV WINDOW CONTRACT
# COLUMN-ROBUST REPAIR
# ======================================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A5-R1 — FD001 30-CYCLE FIT/DEV WINDOW CONTRACT")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Require successful upstream state
# ------------------------------------------------------------------
required_upstream_628hc6a5r1 = [
    "C6_A4_PASSED_628HC6A4",
    "FD001_FIT_RAW_628HC6A3R1",
    "FD001_DEV_RAW_628HC6A3R1",
    "FD001_FIT_TARGET_628HC6A3R1",
    "FD001_DEV_TARGET_628HC6A3R1",
    "FD001_FIT_NORMALIZED_FEATURES_628HC6A4",
    "FD001_DEV_NORMALIZED_FEATURES_628HC6A4",
    "FD001_MODEL_FEATURES_628HC6A4",
]

missing_upstream_628hc6a5r1 = [
    name
    for name in required_upstream_628hc6a5r1
    if name not in globals()
]

if missing_upstream_628hc6a5r1:
    raise RuntimeError(
        "Missing required upstream objects: "
        + ", ".join(missing_upstream_628hc6a5r1)
    )

if not bool(C6_A4_PASSED_628HC6A4):
    raise RuntimeError(
        "C6-A4 must pass before C6-A5-R1."
    )

# ------------------------------------------------------------------
# 1. Load in-memory FIT/DEV objects only
# ------------------------------------------------------------------
WINDOW_LENGTH_628HC6A5R1 = 30
EXPECTED_FEATURE_COUNT_628HC6A5R1 = 17
EXPECTED_FIT_ENGINES_628HC6A5R1 = 80
EXPECTED_DEV_ENGINES_628HC6A5R1 = 20

fit_raw_628hc6a5r1 = (
    FD001_FIT_RAW_628HC6A3R1
    .copy()
    .reset_index(drop=True)
)

dev_raw_628hc6a5r1 = (
    FD001_DEV_RAW_628HC6A3R1
    .copy()
    .reset_index(drop=True)
)

fit_x_rows_628hc6a5r1 = np.asarray(
    FD001_FIT_NORMALIZED_FEATURES_628HC6A4,
    dtype=np.float64,
)

dev_x_rows_628hc6a5r1 = np.asarray(
    FD001_DEV_NORMALIZED_FEATURES_628HC6A4,
    dtype=np.float64,
)

fit_y_rows_628hc6a5r1 = np.asarray(
    FD001_FIT_TARGET_628HC6A3R1,
    dtype=np.float64,
).reshape(-1)

dev_y_rows_628hc6a5r1 = np.asarray(
    FD001_DEV_TARGET_628HC6A3R1,
    dtype=np.float64,
).reshape(-1)

model_features_628hc6a5r1 = list(
    FD001_MODEL_FEATURES_628HC6A4
)

# ------------------------------------------------------------------
# 2. Resolve actual identifier columns safely
# ------------------------------------------------------------------
def resolve_shared_column_628hc6a5r1(
    fit_df,
    dev_df,
    candidates,
    semantic_name,
):
    fit_matches = [
        c for c in candidates
        if c in fit_df.columns
    ]

    dev_matches = [
        c for c in candidates
        if c in dev_df.columns
    ]

    shared = [
        c for c in fit_matches
        if c in dev_matches
    ]

    if len(shared) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {semantic_name} column. "
            f"FIT matches={fit_matches}, "
            f"DEV matches={dev_matches}, "
            f"shared={shared}. "
            f"FIT columns={list(fit_df.columns)}"
        )

    return shared[0]


engine_col_628hc6a5r1 = resolve_shared_column_628hc6a5r1(
    fit_raw_628hc6a5r1,
    dev_raw_628hc6a5r1,
    candidates=[
        "engine_id",
        "unit",
        "unit_id",
        "unit_number",
        "engine",
        "engine_number",
    ],
    semantic_name="engine/unit identifier",
)

cycle_col_628hc6a5r1 = resolve_shared_column_628hc6a5r1(
    fit_raw_628hc6a5r1,
    dev_raw_628hc6a5r1,
    candidates=[
        "cycle",
        "cycle_id",
        "cycle_number",
        "time",
        "time_cycle",
    ],
    semantic_name="cycle identifier",
)

print("\n--- RESOLVED IDENTIFIER COLUMNS ---")
print("Engine/unit column :", engine_col_628hc6a5r1)
print("Cycle column       :", cycle_col_628hc6a5r1)

# ------------------------------------------------------------------
# 3. Structural alignment checks
# ------------------------------------------------------------------
if fit_x_rows_628hc6a5r1.shape != (
    len(fit_raw_628hc6a5r1),
    EXPECTED_FEATURE_COUNT_628HC6A5R1,
):
    raise RuntimeError(
        "FIT normalized-row shape mismatch: "
        f"raw rows={len(fit_raw_628hc6a5r1)}, "
        f"normalized shape={fit_x_rows_628hc6a5r1.shape}"
    )

if dev_x_rows_628hc6a5r1.shape != (
    len(dev_raw_628hc6a5r1),
    EXPECTED_FEATURE_COUNT_628HC6A5R1,
):
    raise RuntimeError(
        "DEV normalized-row shape mismatch: "
        f"raw rows={len(dev_raw_628hc6a5r1)}, "
        f"normalized shape={dev_x_rows_628hc6a5r1.shape}"
    )

if fit_y_rows_628hc6a5r1.shape != (
    len(fit_raw_628hc6a5r1),
):
    raise RuntimeError(
        "FIT target-row shape mismatch: "
        f"raw rows={len(fit_raw_628hc6a5r1)}, "
        f"target shape={fit_y_rows_628hc6a5r1.shape}"
    )

if dev_y_rows_628hc6a5r1.shape != (
    len(dev_raw_628hc6a5r1),
):
    raise RuntimeError(
        "DEV target-row shape mismatch: "
        f"raw rows={len(dev_raw_628hc6a5r1)}, "
        f"target shape={dev_y_rows_628hc6a5r1.shape}"
    )

if len(model_features_628hc6a5r1) != 17:
    raise RuntimeError(
        "Expected exactly 17 locked model features, "
        f"got {len(model_features_628hc6a5r1)}."
    )

# ------------------------------------------------------------------
# 4. Verify engine/cycle structure before windowing
# ------------------------------------------------------------------
def validate_engine_cycle_structure_628hc6a5r1(
    raw_df,
    engine_col,
    cycle_col,
    population_name,
):
    engines = raw_df[
        engine_col
    ].to_numpy()

    cycles = raw_df[
        cycle_col
    ].to_numpy()

    unique_engines = pd.unique(
        engines
    )

    lengths = []

    for engine_id in unique_engines:
        idx = np.flatnonzero(
            engines == engine_id
        )

        engine_cycles = cycles[
            idx
        ]

        if idx.size == 0:
            raise RuntimeError(
                f"{population_name}: empty engine group."
            )

        if not np.all(
            np.diff(idx) == 1
        ):
            raise RuntimeError(
                f"{population_name}: engine {engine_id} "
                "rows are not contiguous."
            )

        if not np.all(
            np.diff(engine_cycles) == 1
        ):
            raise RuntimeError(
                f"{population_name}: engine {engine_id} "
                "cycles are not consecutive."
            )

        lengths.append(
            idx.size
        )

    lengths = np.asarray(
        lengths,
        dtype=np.int64,
    )

    if int(lengths.min()) < WINDOW_LENGTH_628HC6A5R1:
        raise RuntimeError(
            f"{population_name}: an engine has fewer than "
            f"{WINDOW_LENGTH_628HC6A5R1} cycles."
        )

    return unique_engines, lengths


(
    fit_engines_628hc6a5r1,
    fit_lengths_628hc6a5r1,
) = validate_engine_cycle_structure_628hc6a5r1(
    fit_raw_628hc6a5r1,
    engine_col_628hc6a5r1,
    cycle_col_628hc6a5r1,
    "FIT",
)

(
    dev_engines_628hc6a5r1,
    dev_lengths_628hc6a5r1,
) = validate_engine_cycle_structure_628hc6a5r1(
    dev_raw_628hc6a5r1,
    engine_col_628hc6a5r1,
    cycle_col_628hc6a5r1,
    "DEV",
)

# ------------------------------------------------------------------
# 5. Build 30-cycle sliding windows
# ------------------------------------------------------------------
def build_windows_628hc6a5r1(
    raw_df,
    normalized_rows,
    target_rows,
    engine_col,
    cycle_col,
    window_length,
):
    x_rows = np.asarray(
        normalized_rows,
        dtype=np.float64,
    )

    y_rows = np.asarray(
        target_rows,
        dtype=np.float64,
    ).reshape(-1)

    engine_values = raw_df[
        engine_col
    ].to_numpy()

    cycle_values = raw_df[
        cycle_col
    ].to_numpy()

    windows = []
    targets = []

    engine_ids = []
    start_cycles = []
    end_cycles = []
    endpoint_rows = []

    for engine_id in pd.unique(
        engine_values
    ):
        idx = np.flatnonzero(
            engine_values == engine_id
        )

        for local_end in range(
            window_length - 1,
            idx.size,
        ):
            local_start = (
                local_end
                - window_length
                + 1
            )

            row_idx = idx[
                local_start:
                local_end + 1
            ]

            endpoint_row = int(
                idx[local_end]
            )

            windows.append(
                x_rows[
                    row_idx,
                    :
                ]
            )

            targets.append(
                y_rows[
                    endpoint_row
                ]
            )

            engine_ids.append(
                int(engine_id)
            )

            start_cycles.append(
                int(
                    cycle_values[
                        row_idx[0]
                    ]
                )
            )

            end_cycles.append(
                int(
                    cycle_values[
                        endpoint_row
                    ]
                )
            )

            endpoint_rows.append(
                endpoint_row
            )

    X = np.stack(
        windows,
        axis=0,
    ).astype(
        np.float64,
        copy=False,
    )

    y = np.asarray(
        targets,
        dtype=np.float64,
    )

    metadata = pd.DataFrame(
        {
            "engine_id": np.asarray(
                engine_ids,
                dtype=np.int64,
            ),
            "start_cycle": np.asarray(
                start_cycles,
                dtype=np.int64,
            ),
            "end_cycle": np.asarray(
                end_cycles,
                dtype=np.int64,
            ),
            "endpoint_row_position": np.asarray(
                endpoint_rows,
                dtype=np.int64,
            ),
        }
    )

    return X, y, metadata


(
    FD001_X_FIT_WINDOWS_628HC6A5R1,
    FD001_Y_FIT_WINDOWS_628HC6A5R1,
    FD001_FIT_WINDOW_METADATA_628HC6A5R1,
) = build_windows_628hc6a5r1(
    fit_raw_628hc6a5r1,
    fit_x_rows_628hc6a5r1,
    fit_y_rows_628hc6a5r1,
    engine_col_628hc6a5r1,
    cycle_col_628hc6a5r1,
    WINDOW_LENGTH_628HC6A5R1,
)

(
    FD001_X_DEV_WINDOWS_628HC6A5R1,
    FD001_Y_DEV_WINDOWS_628HC6A5R1,
    FD001_DEV_WINDOW_METADATA_628HC6A5R1,
) = build_windows_628hc6a5r1(
    dev_raw_628hc6a5r1,
    dev_x_rows_628hc6a5r1,
    dev_y_rows_628hc6a5r1,
    engine_col_628hc6a5r1,
    cycle_col_628hc6a5r1,
    WINDOW_LENGTH_628HC6A5R1,
)

# ------------------------------------------------------------------
# 6. Expected counts
# ------------------------------------------------------------------
EXPECTED_FIT_WINDOWS_628HC6A5R1 = int(
    np.sum(
        fit_lengths_628hc6a5r1
        - WINDOW_LENGTH_628HC6A5R1
        + 1
    )
)

EXPECTED_DEV_WINDOWS_628HC6A5R1 = int(
    np.sum(
        dev_lengths_628hc6a5r1
        - WINDOW_LENGTH_628HC6A5R1
        + 1
    )
)

# ------------------------------------------------------------------
# 7. Exact endpoint-target alignment
# ------------------------------------------------------------------
fit_endpoint_rows_628hc6a5r1 = (
    FD001_FIT_WINDOW_METADATA_628HC6A5R1[
        "endpoint_row_position"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

dev_endpoint_rows_628hc6a5r1 = (
    FD001_DEV_WINDOW_METADATA_628HC6A5R1[
        "endpoint_row_position"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

fit_endpoint_targets_628hc6a5r1 = (
    fit_y_rows_628hc6a5r1[
        fit_endpoint_rows_628hc6a5r1
    ]
)

dev_endpoint_targets_628hc6a5r1 = (
    dev_y_rows_628hc6a5r1[
        dev_endpoint_rows_628hc6a5r1
    ]
)

# ------------------------------------------------------------------
# 8. SHA-256
# ------------------------------------------------------------------
def sha256_array_628hc6a5r1(arr):
    a = np.ascontiguousarray(
        arr
    )

    return hashlib.sha256(
        a.tobytes(
            order="C"
        )
    ).hexdigest()


FD001_X_FIT_WINDOWS_SHA256_628HC6A5R1 = (
    sha256_array_628hc6a5r1(
        FD001_X_FIT_WINDOWS_628HC6A5R1
    )
)

FD001_Y_FIT_WINDOWS_SHA256_628HC6A5R1 = (
    sha256_array_628hc6a5r1(
        FD001_Y_FIT_WINDOWS_628HC6A5R1
    )
)

FD001_X_DEV_WINDOWS_SHA256_628HC6A5R1 = (
    sha256_array_628hc6a5r1(
        FD001_X_DEV_WINDOWS_628HC6A5R1
    )
)

FD001_Y_DEV_WINDOWS_SHA256_628HC6A5R1 = (
    sha256_array_628hc6a5r1(
        FD001_Y_DEV_WINDOWS_628HC6A5R1
    )
)

# ------------------------------------------------------------------
# 9. Contract
# ------------------------------------------------------------------
FD001_WINDOW_CONTRACT_628HC6A5R1 = {
    "dataset": "FD001",
    "window_length": 30,
    "feature_count": 17,
    "engine_column_source":
        engine_col_628hc6a5r1,
    "cycle_column_source":
        cycle_col_628hc6a5r1,
    "model_features":
        tuple(
            model_features_628hc6a5r1
        ),
    "fit_window_count":
        int(
            FD001_X_FIT_WINDOWS_628HC6A5R1.shape[
                0
            ]
        ),
    "dev_window_count":
        int(
            FD001_X_DEV_WINDOWS_628HC6A5R1.shape[
                0
            ]
        ),
    "fit_shape":
        tuple(
            FD001_X_FIT_WINDOWS_628HC6A5R1.shape
        ),
    "dev_shape":
        tuple(
            FD001_X_DEV_WINDOWS_628HC6A5R1.shape
        ),
    "target_semantics":
        "capped RUL at window endpoint",
    "whole_engine_split":
        True,
    "fit_only_normalization_reference":
        True,
    "test_features_used":
        False,
    "official_test_labels_used":
        False,
    "model_fitted_here":
        False,
    "threshold_selected_here":
        False,
}

# ------------------------------------------------------------------
# 10. Audit
# ------------------------------------------------------------------
fit_engine_set_628hc6a5r1 = set(
    FD001_FIT_WINDOW_METADATA_628HC6A5R1[
        "engine_id"
    ]
    .unique()
    .tolist()
)

dev_engine_set_628hc6a5r1 = set(
    FD001_DEV_WINDOW_METADATA_628HC6A5R1[
        "engine_id"
    ]
    .unique()
    .tolist()
)

audit_rows_628hc6a5r1 = [
    (
        "c6_a4_passed",
        bool(
            C6_A4_PASSED_628HC6A4
        ),
    ),
    (
        "resolved_engine_column",
        isinstance(
            engine_col_628hc6a5r1,
            str,
        ),
    ),
    (
        "resolved_cycle_column",
        isinstance(
            cycle_col_628hc6a5r1,
            str,
        ),
    ),
    (
        "window_length_30",
        WINDOW_LENGTH_628HC6A5R1 == 30,
    ),
    (
        "feature_count_17",
        len(
            model_features_628hc6a5r1
        ) == 17,
    ),
    (
        "fit_engine_count_80",
        len(
            fit_engines_628hc6a5r1
        ) == 80,
    ),
    (
        "dev_engine_count_20",
        len(
            dev_engines_628hc6a5r1
        ) == 20,
    ),
    (
        "fit_dev_engine_disjoint",
        fit_engine_set_628hc6a5r1.isdisjoint(
            dev_engine_set_628hc6a5r1
        ),
    ),
    (
        "fit_window_count_14459",
        EXPECTED_FIT_WINDOWS_628HC6A5R1
        == 14459,
    ),
    (
        "dev_window_count_3272",
        EXPECTED_DEV_WINDOWS_628HC6A5R1
        == 3272,
    ),
    (
        "fit_window_shape_14459x30x17",
        FD001_X_FIT_WINDOWS_628HC6A5R1.shape
        == (
            14459,
            30,
            17,
        ),
    ),
    (
        "dev_window_shape_3272x30x17",
        FD001_X_DEV_WINDOWS_628HC6A5R1.shape
        == (
            3272,
            30,
            17,
        ),
    ),
    (
        "fit_target_shape_14459",
        FD001_Y_FIT_WINDOWS_628HC6A5R1.shape
        == (
            14459,
        ),
    ),
    (
        "dev_target_shape_3272",
        FD001_Y_DEV_WINDOWS_628HC6A5R1.shape
        == (
            3272,
        ),
    ),
    (
        "fit_windows_finite",
        bool(
            np.isfinite(
                FD001_X_FIT_WINDOWS_628HC6A5R1
            ).all()
        ),
    ),
    (
        "dev_windows_finite",
        bool(
            np.isfinite(
                FD001_X_DEV_WINDOWS_628HC6A5R1
            ).all()
        ),
    ),
    (
        "fit_targets_finite",
        bool(
            np.isfinite(
                FD001_Y_FIT_WINDOWS_628HC6A5R1
            ).all()
        ),
    ),
    (
        "dev_targets_finite",
        bool(
            np.isfinite(
                FD001_Y_DEV_WINDOWS_628HC6A5R1
            ).all()
        ),
    ),
    (
        "fit_targets_within_0_125",
        bool(
            np.all(
                (
                    FD001_Y_FIT_WINDOWS_628HC6A5R1
                    >= 0.0
                )
                &
                (
                    FD001_Y_FIT_WINDOWS_628HC6A5R1
                    <= 125.0
                )
            )
        ),
    ),
    (
        "dev_targets_within_0_125",
        bool(
            np.all(
                (
                    FD001_Y_DEV_WINDOWS_628HC6A5R1
                    >= 0.0
                )
                &
                (
                    FD001_Y_DEV_WINDOWS_628HC6A5R1
                    <= 125.0
                )
            )
        ),
    ),
    (
        "fit_endpoint_targets_exact",
        bool(
            np.array_equal(
                FD001_Y_FIT_WINDOWS_628HC6A5R1,
                fit_endpoint_targets_628hc6a5r1,
            )
        ),
    ),
    (
        "dev_endpoint_targets_exact",
        bool(
            np.array_equal(
                FD001_Y_DEV_WINDOWS_628HC6A5R1,
                dev_endpoint_targets_628hc6a5r1,
            )
        ),
    ),
    (
        "fit_window_span_exact_30",
        bool(
            np.all(
                (
                    FD001_FIT_WINDOW_METADATA_628HC6A5R1[
                        "end_cycle"
                    ].to_numpy()
                    -
                    FD001_FIT_WINDOW_METADATA_628HC6A5R1[
                        "start_cycle"
                    ].to_numpy()
                )
                == 29
            )
        ),
    ),
    (
        "dev_window_span_exact_30",
        bool(
            np.all(
                (
                    FD001_DEV_WINDOW_METADATA_628HC6A5R1[
                        "end_cycle"
                    ].to_numpy()
                    -
                    FD001_DEV_WINDOW_METADATA_628HC6A5R1[
                        "start_cycle"
                    ].to_numpy()
                )
                == 29
            )
        ),
    ),
    (
        "test_features_not_read",
        True,
    ),
    (
        "official_test_labels_not_read",
        True,
    ),
    (
        "no_model_execution",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning",
        True,
    ),
    (
        "no_threshold_modification",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

C6_A5_R1_AUDIT_628HC6A5R1 = pd.DataFrame(
    audit_rows_628hc6a5r1,
    columns=[
        "check",
        "passed",
    ],
)

C6_A5_R1_PASSED_628HC6A5R1 = bool(
    C6_A5_R1_AUDIT_628HC6A5R1[
        "passed"
    ].all()
)

# ------------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------------
print(
    "\n--- WINDOW COUNT CONTRACT ---"
)

print(
    "Expected FIT windows :",
    EXPECTED_FIT_WINDOWS_628HC6A5R1,
)

print(
    "Actual FIT windows   :",
    FD001_X_FIT_WINDOWS_628HC6A5R1.shape[
        0
    ],
)

print(
    "Expected DEV windows :",
    EXPECTED_DEV_WINDOWS_628HC6A5R1,
)

print(
    "Actual DEV windows   :",
    FD001_X_DEV_WINDOWS_628HC6A5R1.shape[
        0
    ],
)

print(
    "\n--- WINDOW SHAPES ---"
)

print(
    "FIT X / y :",
    FD001_X_FIT_WINDOWS_628HC6A5R1.shape,
    FD001_Y_FIT_WINDOWS_628HC6A5R1.shape,
)

print(
    "DEV X / y :",
    FD001_X_DEV_WINDOWS_628HC6A5R1.shape,
    FD001_Y_DEV_WINDOWS_628HC6A5R1.shape,
)

print(
    "\n--- FIRST 10 FIT WINDOW ENDPOINTS ---"
)

display(
    FD001_FIT_WINDOW_METADATA_628HC6A5R1.head(
        10
    )
)

print(
    "\n--- FIRST 10 DEV WINDOW ENDPOINTS ---"
)

display(
    FD001_DEV_WINDOW_METADATA_628HC6A5R1.head(
        10
    )
)

print(
    "\n--- SHA-256 ---"
)

print(
    "FIT X :",
    FD001_X_FIT_WINDOWS_SHA256_628HC6A5R1,
)

print(
    "FIT y :",
    FD001_Y_FIT_WINDOWS_SHA256_628HC6A5R1,
)

print(
    "DEV X :",
    FD001_X_DEV_WINDOWS_SHA256_628HC6A5R1,
)

print(
    "DEV y :",
    FD001_Y_DEV_WINDOWS_SHA256_628HC6A5R1,
)

print(
    "\n--- C6-A5-R1 AUDIT ---"
)

display(
    C6_A5_R1_AUDIT_628HC6A5R1
)

# ------------------------------------------------------------------
# 12. Fail safely
# ------------------------------------------------------------------
if not C6_A5_R1_PASSED_628HC6A5R1:
    failed_628hc6a5r1 = (
        C6_A5_R1_AUDIT_628HC6A5R1.loc[
            ~C6_A5_R1_AUDIT_628HC6A5R1[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A5-R1 window contract failed. Failed checks: "
        + ", ".join(
            failed_628hc6a5r1
        )
    )

# ------------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------------
print(
    "\n"
    + "=" * 118
)

print(
    "STEP 6.28H-C6-A5-R1 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "Resolved engine/unit column       :",
    engine_col_628hc6a5r1,
)

print(
    "Resolved cycle column             :",
    cycle_col_628hc6a5r1,
)

print(
    "Window length                     :",
    WINDOW_LENGTH_628HC6A5R1,
)

print(
    "Locked feature count              :",
    len(
        model_features_628hc6a5r1
    ),
)

print(
    "FIT windows                       :",
    FD001_X_FIT_WINDOWS_628HC6A5R1.shape,
)

print(
    "DEV windows                       :",
    FD001_X_DEV_WINDOWS_628HC6A5R1.shape,
)

print(
    "FIT / DEV engines                 :",
    len(
        fit_engines_628hc6a5r1
    ),
    "/",
    len(
        dev_engines_628hc6a5r1
    ),
)

print(
    "Whole-engine leakage prevented    :",
    fit_engine_set_628hc6a5r1.isdisjoint(
        dev_engine_set_628hc6a5r1
    ),
)

print(
    "Endpoint target alignment exact   :",
    bool(
        np.array_equal(
            FD001_Y_FIT_WINDOWS_628HC6A5R1,
            fit_endpoint_targets_628hc6a5r1,
        )
        and
        np.array_equal(
            FD001_Y_DEV_WINDOWS_628HC6A5R1,
            dev_endpoint_targets_628hc6a5r1,
        )
    ),
)

print(
    "TEST features read                :",
    False,
)

print(
    "Official FD001 TEST labels read   :",
    False,
)

print(
    "Model executed/retrained          :",
    False,
)

print(
    "Threshold tuned/modified          :",
    False,
)

print(
    "Files written                     :",
    False,
)

print(
    "All C6-A5-R1 checks passed        :",
    C6_A5_R1_PASSED_628HC6A5R1,
)

print(
    "\nNext required step: C6-A6 — establish the frozen "
    "FD001 model-training contract and train on FIT windows only, "
    "with DEV used only for permitted diagnostics/model selection."
)

print(
    "\n=== END STEP 6.28H-C6-A5-R1 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A6-A0 — FD001 MODEL-TRAINING CONTRACT RECOVERY / TRANSFER LOCK
# ======================================================================================

import inspect
from pprint import pprint

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A6-A0 — FD001 MODEL-TRAINING CONTRACT RECOVERY / TRANSFER LOCK")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Require the completed FD001 window contract
# ------------------------------------------------------------------
if "C6_A5_R1_PASSED_628HC6A5R1" not in globals():
    raise RuntimeError(
        "C6-A5-R1 completion marker is missing. "
        "Run the successful C6-A5-R1 cell before C6-A6-A0."
    )

if not bool(C6_A5_R1_PASSED_628HC6A5R1):
    raise RuntimeError(
        "C6-A5-R1 must pass before establishing the FD001 model-training contract."
    )

required_fd001_objects_628hc6a6a0 = [
    "FD001_X_FIT_WINDOWS_628HC6A5R1",
    "FD001_Y_FIT_WINDOWS_628HC6A5R1",
    "FD001_X_DEV_WINDOWS_628HC6A5R1",
    "FD001_Y_DEV_WINDOWS_628HC6A5R1",
    "FD001_MODEL_FEATURES_628HC6A4",
]

missing_fd001_objects_628hc6a6a0 = [
    name
    for name in required_fd001_objects_628hc6a6a0
    if name not in globals()
]

if missing_fd001_objects_628hc6a6a0:
    raise RuntimeError(
        "Missing required FD001 objects: "
        + ", ".join(missing_fd001_objects_628hc6a6a0)
    )

# ------------------------------------------------------------------
# 1. Require frozen FD002 model-development evidence
# ------------------------------------------------------------------
required_fd002_contract_objects_628hc6a6a0 = [
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "FD002_MODEL_EVIDENCE_628EB",
]

missing_fd002_contract_objects_628hc6a6a0 = [
    name
    for name in required_fd002_contract_objects_628hc6a6a0
    if name not in globals()
]

if missing_fd002_contract_objects_628hc6a6a0:
    raise RuntimeError(
        "Frozen FD002 model-contract evidence is missing: "
        + ", ".join(missing_fd002_contract_objects_628hc6a6a0)
        + ". Do not guess FD001 architecture/hyperparameters."
    )

# ------------------------------------------------------------------
# 2. Validate current FD001 tensors
# ------------------------------------------------------------------
x_fit_628hc6a6a0 = np.asarray(
    FD001_X_FIT_WINDOWS_628HC6A5R1,
    dtype=np.float64,
)

y_fit_628hc6a6a0 = np.asarray(
    FD001_Y_FIT_WINDOWS_628HC6A5R1,
    dtype=np.float64,
).reshape(-1)

x_dev_628hc6a6a0 = np.asarray(
    FD001_X_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float64,
)

y_dev_628hc6a6a0 = np.asarray(
    FD001_Y_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float64,
).reshape(-1)

if x_fit_628hc6a6a0.shape != (14459, 30, 17):
    raise RuntimeError(
        f"Unexpected FD001 FIT window shape: {x_fit_628hc6a6a0.shape}"
    )

if x_dev_628hc6a6a0.shape != (3272, 30, 17):
    raise RuntimeError(
        f"Unexpected FD001 DEV window shape: {x_dev_628hc6a6a0.shape}"
    )

if y_fit_628hc6a6a0.shape != (14459,):
    raise RuntimeError(
        f"Unexpected FD001 FIT target shape: {y_fit_628hc6a6a0.shape}"
    )

if y_dev_628hc6a6a0.shape != (3272,):
    raise RuntimeError(
        f"Unexpected FD001 DEV target shape: {y_dev_628hc6a6a0.shape}"
    )

if not (
    np.isfinite(x_fit_628hc6a6a0).all()
    and np.isfinite(y_fit_628hc6a6a0).all()
    and np.isfinite(x_dev_628hc6a6a0).all()
    and np.isfinite(y_dev_628hc6a6a0).all()
):
    raise RuntimeError(
        "Non-finite FD001 FIT/DEV training data detected."
    )

# ------------------------------------------------------------------
# 3. Recover exact frozen FD002 contract/evidence
# ------------------------------------------------------------------
fd002_training_contract_628hc6a6a0 = (
    FD002_GRU_TRAINING_CONTRACT_628EA
)

fd002_model_evidence_628hc6a6a0 = (
    FD002_MODEL_EVIDENCE_628EB
)

print("\n--- FROZEN FD002 GRU TRAINING CONTRACT ---")
pprint(
    fd002_training_contract_628hc6a6a0,
    sort_dicts=False,
)

print("\n--- FROZEN FD002 MODEL EVIDENCE ---")
pprint(
    fd002_model_evidence_628hc6a6a0,
    sort_dicts=False,
)

# ------------------------------------------------------------------
# 4. Flatten nested mappings for explicit inspection
# ------------------------------------------------------------------
def flatten_mapping_628hc6a6a0(obj, prefix=""):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if isinstance(value, dict):
                rows.extend(
                    flatten_mapping_628hc6a6a0(
                        value,
                        prefix=new_prefix,
                    )
                )
            else:
                rows.append(
                    {
                        "key_path": new_prefix,
                        "type": type(value).__name__,
                        "value_repr": repr(value),
                    }
                )

    else:
        rows.append(
            {
                "key_path": prefix or "<root>",
                "type": type(obj).__name__,
                "value_repr": repr(obj),
            }
        )

    return rows


FD002_GRU_TRAINING_CONTRACT_FLAT_628HC6A6A0 = (
    pd.DataFrame(
        flatten_mapping_628hc6a6a0(
            fd002_training_contract_628hc6a6a0
        )
    )
)

FD002_MODEL_EVIDENCE_FLAT_628HC6A6A0 = (
    pd.DataFrame(
        flatten_mapping_628hc6a6a0(
            fd002_model_evidence_628hc6a6a0
        )
    )
)

print("\n--- FLATTENED FD002 TRAINING CONTRACT ---")
display(
    FD002_GRU_TRAINING_CONTRACT_FLAT_628HC6A6A0
)

print("\n--- FLATTENED FD002 MODEL EVIDENCE ---")
display(
    FD002_MODEL_EVIDENCE_FLAT_628HC6A6A0
)

# ------------------------------------------------------------------
# 5. Inventory existing PyTorch models/classes safely
#    NO forward pass and NO training here.
# ------------------------------------------------------------------
torch_available_628hc6a6a0 = False
torch_model_rows_628hc6a6a0 = []
torch_class_rows_628hc6a6a0 = []

try:
    import torch

    torch_available_628hc6a6a0 = True

    globals_snapshot_628hc6a6a0 = list(
        globals().items()
    )

    for name, obj in globals_snapshot_628hc6a6a0:

        if isinstance(obj, torch.nn.Module):

            param_count = int(
                sum(
                    p.numel()
                    for p in obj.parameters()
                )
            )

            trainable_count = int(
                sum(
                    p.numel()
                    for p in obj.parameters()
                    if p.requires_grad
                )
            )

            state_shapes = {
                key: tuple(value.shape)
                for key, value
                in obj.state_dict().items()
            }

            torch_model_rows_628hc6a6a0.append(
                {
                    "name": name,
                    "class_name": type(obj).__name__,
                    "module": type(obj).__module__,
                    "parameter_count": param_count,
                    "trainable_parameter_count":
                        trainable_count,
                    "training_mode":
                        bool(obj.training),
                    "state_dict_shapes":
                        repr(state_shapes),
                }
            )

        if inspect.isclass(obj):

            try:
                is_torch_module_class = issubclass(
                    obj,
                    torch.nn.Module,
                )
            except TypeError:
                is_torch_module_class = False

            if (
                is_torch_module_class
                and obj is not torch.nn.Module
            ):
                try:
                    signature = str(
                        inspect.signature(obj)
                    )
                except (TypeError, ValueError):
                    signature = (
                        "<signature unavailable>"
                    )

                torch_class_rows_628hc6a6a0.append(
                    {
                        "name": name,
                        "module": getattr(
                            obj,
                            "__module__",
                            "",
                        ),
                        "signature": signature,
                    }
                )

except ImportError:
    pass


FD002_EXISTING_TORCH_MODELS_628HC6A6A0 = (
    pd.DataFrame(
        torch_model_rows_628hc6a6a0
    )
)

FD002_EXISTING_TORCH_MODEL_CLASSES_628HC6A6A0 = (
    pd.DataFrame(
        torch_class_rows_628hc6a6a0
    )
)

print("\n--- EXISTING PYTORCH MODEL INSTANCES ---")

if len(
    FD002_EXISTING_TORCH_MODELS_628HC6A6A0
) == 0:
    print("None found.")
else:
    display(
        FD002_EXISTING_TORCH_MODELS_628HC6A6A0
    )

print("\n--- EXISTING PYTORCH MODEL CLASSES ---")

if len(
    FD002_EXISTING_TORCH_MODEL_CLASSES_628HC6A6A0
) == 0:
    print("None found.")
else:
    display(
        FD002_EXISTING_TORCH_MODEL_CLASSES_628HC6A6A0
    )

# ------------------------------------------------------------------
# 6. Inventory relevant model/training helper functions
# ------------------------------------------------------------------
helper_keywords_628hc6a6a0 = (
    "gru",
    "train",
    "model",
    "early",
    "loss",
)

helper_rows_628hc6a6a0 = []

for name, obj in list(
    globals().items()
):

    if not inspect.isfunction(obj):
        continue

    name_lower = name.lower()

    if not any(
        token in name_lower
        for token in helper_keywords_628hc6a6a0
    ):
        continue

    try:
        signature = str(
            inspect.signature(obj)
        )
    except (TypeError, ValueError):
        signature = "<signature unavailable>"

    helper_rows_628hc6a6a0.append(
        {
            "name": name,
            "signature": signature,
            "module": getattr(
                obj,
                "__module__",
                "",
            ),
        }
    )


FD001_MODEL_TRAINING_HELPER_INVENTORY_628HC6A6A0 = (
    pd.DataFrame(
        helper_rows_628hc6a6a0
    )
)

print(
    "\n--- RELEVANT EXISTING TRAINING / MODEL HELPERS ---"
)

if len(
    FD001_MODEL_TRAINING_HELPER_INVENTORY_628HC6A6A0
) == 0:
    print("None found.")
else:
    display(
        FD001_MODEL_TRAINING_HELPER_INVENTORY_628HC6A6A0
        .sort_values("name")
        .reset_index(drop=True)
    )

# ------------------------------------------------------------------
# 7. Lock the FD001 transfer semantics
# ------------------------------------------------------------------
FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0 = {
    "dataset": "FD001",

    "research_problem":
        "Reliability-Aware Selective Remaining Useful Life "
        "Prediction under Sensor Degradation",

    "source_reference_subset": "FD002",

    "window_length": 30,

    "input_feature_count": 17,

    "fit_window_count": 14459,

    "dev_window_count": 3272,

    "target":
        "piecewise-capped RUL, cap=125",

    "training_population":
        "FD001 FIT windows only",

    "development_population":
        "FD001 DEV windows only for permitted "
        "diagnostics/model selection",

    "point_model_transfer_rule":
        "reuse frozen FD002 architecture/hyperparameter logic; "
        "refit weights on FD001 FIT",

    "reliability_architecture":
        "uncertainty_only",

    "mc_dropout_passes":
        50,

    "numeric_risk_threshold":
        "derive later from FD001 CLEAN DEV only, then freeze",

    "test_features_used_here":
        False,

    "official_test_labels_used_here":
        False,

    "model_executed_here":
        False,

    "model_retrained_here":
        False,

    "threshold_tuned_here":
        False,

    "files_written_here":
        False,
}

print("\n--- FD001 TRANSFER TRAINING CONTRACT ---")

for key, value in (
    FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0.items()
):
    print(
        f"{key:<34}: {value}"
    )

# ------------------------------------------------------------------
# 8. Audit
# ------------------------------------------------------------------
audit_rows_628hc6a6a0 = [
    (
        "c6_a5_r1_passed",
        bool(
            C6_A5_R1_PASSED_628HC6A5R1
        ),
    ),
    (
        "fd001_fit_shape_14459x30x17",
        x_fit_628hc6a6a0.shape
        == (14459, 30, 17),
    ),
    (
        "fd001_dev_shape_3272x30x17",
        x_dev_628hc6a6a0.shape
        == (3272, 30, 17),
    ),
    (
        "fd001_fit_target_shape_14459",
        y_fit_628hc6a6a0.shape
        == (14459,),
    ),
    (
        "fd001_dev_target_shape_3272",
        y_dev_628hc6a6a0.shape
        == (3272,),
    ),
    (
        "fd001_training_data_finite",
        bool(
            np.isfinite(
                x_fit_628hc6a6a0
            ).all()
            and np.isfinite(
                y_fit_628hc6a6a0
            ).all()
            and np.isfinite(
                x_dev_628hc6a6a0
            ).all()
            and np.isfinite(
                y_dev_628hc6a6a0
            ).all()
        ),
    ),
    (
        "fd002_training_contract_present",
        "FD002_GRU_TRAINING_CONTRACT_628EA"
        in globals(),
    ),
    (
        "fd002_model_evidence_present",
        "FD002_MODEL_EVIDENCE_628EB"
        in globals(),
    ),
    (
        "fd002_training_contract_is_mapping",
        isinstance(
            fd002_training_contract_628hc6a6a0,
            dict,
        ),
    ),
    (
        "fd002_model_evidence_is_mapping",
        isinstance(
            fd002_model_evidence_628hc6a6a0,
            dict,
        ),
    ),
    (
        "window_length_locked_30",
        FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0[
            "window_length"
        ] == 30,
    ),
    (
        "feature_count_locked_17",
        FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0[
            "input_feature_count"
        ] == 17,
    ),
    (
        "reliability_architecture_uncertainty_only",
        FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0[
            "reliability_architecture"
        ] == "uncertainty_only",
    ),
    (
        "mc_dropout_passes_50",
        FD001_MODEL_TRAINING_TRANSFER_CONTRACT_628HC6A6A0[
            "mc_dropout_passes"
        ] == 50,
    ),
    (
        "test_features_not_used",
        True,
    ),
    (
        "official_test_labels_not_used",
        True,
    ),
    (
        "no_model_execution",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

C6_A6_A0_AUDIT_628HC6A6A0 = pd.DataFrame(
    audit_rows_628hc6a6a0,
    columns=[
        "check",
        "passed",
    ],
)

C6_A6_A0_PASSED_628HC6A6A0 = bool(
    C6_A6_A0_AUDIT_628HC6A6A0[
        "passed"
    ].all()
)

print("\n--- C6-A6-A0 AUDIT ---")

display(
    C6_A6_A0_AUDIT_628HC6A6A0
)

if not C6_A6_A0_PASSED_628HC6A6A0:

    failed_628hc6a6a0 = (
        C6_A6_A0_AUDIT_628HC6A6A0.loc[
            ~C6_A6_A0_AUDIT_628HC6A6A0[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A6-A0 contract recovery failed. Failed checks: "
        + ", ".join(
            failed_628hc6a6a0
        )
    )

# ------------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 118)

print(
    "STEP 6.28H-C6-A6-A0 FINAL SUMMARY"
)

print("=" * 118)

print(
    "FD001 FIT windows ready            :",
    x_fit_628hc6a6a0.shape,
)

print(
    "FD001 DEV windows ready            :",
    x_dev_628hc6a6a0.shape,
)

print(
    "FD002 training contract recovered  :",
    True,
)

print(
    "FD002 model evidence recovered     :",
    True,
)

print(
    "PyTorch available                  :",
    torch_available_628hc6a6a0,
)

print(
    "Existing torch model instances     :",
    len(
        FD002_EXISTING_TORCH_MODELS_628HC6A6A0
    ),
)

print(
    "Existing torch model classes       :",
    len(
        FD002_EXISTING_TORCH_MODEL_CLASSES_628HC6A6A0
    ),
)

print(
    "Training/model helpers found       :",
    len(
        FD001_MODEL_TRAINING_HELPER_INVENTORY_628HC6A6A0
    ),
)

print(
    "Official TEST features used        :",
    False,
)

print(
    "Official FD001 TEST labels used    :",
    False,
)

print(
    "Model executed/retrained           :",
    False,
)

print(
    "Threshold tuned/modified           :",
    False,
)

print(
    "Files written                      :",
    False,
)

print(
    "All C6-A6-A0 checks passed         :",
    C6_A6_A0_PASSED_628HC6A6A0,
)

print(
    "\nNext required step: C6-A6-A1 — instantiate the exact "
    "transferred GRU architecture/hyperparameters recovered above "
    "and train FD001 weights using FIT windows only."
)

print(
    "\n=== END STEP 6.28H-C6-A6-A0 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A6-A1 — FD001 TRANSFERRED GRU TRAINING (FIT ONLY)
# ======================================================================================

import copy
import hashlib
import inspect
import random
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

print("=" * 116)
print("STEP 6.28H-C6-A6-A1 — FD001 TRANSFERRED GRU TRAINING (FIT ONLY)")
print("=" * 116)

# ------------------------------------------------------------------
# 0. Hard prerequisites
# ------------------------------------------------------------------
if not bool(globals().get("C6_A6_A0_PASSED_628HC6A6A0", False)):
    raise RuntimeError("C6-A6-A0 must pass before C6-A6-A1.")

required_628hc6a6a1 = [
    "FD001_X_FIT_WINDOWS_628HC6A5R1",
    "FD001_Y_FIT_WINDOWS_628HC6A5R1",
    "FD001_X_DEV_WINDOWS_628HC6A5R1",
    "FD001_Y_DEV_WINDOWS_628HC6A5R1",
    "FD002_GRU_TRAINING_CONTRACT_628EA",
    "FD002GRURegressor628EB",
    "MODEL_FD002_628EB",
]

missing_628hc6a6a1 = [
    n for n in required_628hc6a6a1
    if n not in globals()
]

if missing_628hc6a6a1:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc6a6a1)
    )

# ------------------------------------------------------------------
# 1. Exact frozen contract
# ------------------------------------------------------------------
contract_628hc6a6a1 = dict(
    FD002_GRU_TRAINING_CONTRACT_628EA
)

expected_628hc6a6a1 = {
    "architecture": "single_layer_GRU_regressor",
    "window_length": 30,
    "input_size": 17,
    "hidden_size": 64,
    "num_layers": 1,
    "dense_hidden_size": 32,
    "dropout": 0.2,
    "target": "RUL_cap_125",
    "training_batch_size": 128,
    "validation_batch_size": 256,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "MSELoss",
    "gradient_clip_max_norm": 1.0,
    "max_epochs": 100,
    "patience": 12,
    "seed": 42,
}

bad_contract_628hc6a6a1 = {
    k: (
        contract_628hc6a6a1.get(k),
        v,
    )
    for k, v in expected_628hc6a6a1.items()
    if contract_628hc6a6a1.get(k) != v
}

if bad_contract_628hc6a6a1:
    raise RuntimeError(
        "Frozen training contract mismatch: "
        + repr(bad_contract_628hc6a6a1)
    )

# ------------------------------------------------------------------
# 2. FIT / DEV data only
# ------------------------------------------------------------------
x_fit_628hc6a6a1 = np.asarray(
    FD001_X_FIT_WINDOWS_628HC6A5R1,
    dtype=np.float32,
)

y_fit_628hc6a6a1 = np.asarray(
    FD001_Y_FIT_WINDOWS_628HC6A5R1,
    dtype=np.float32,
).reshape(-1)

x_dev_628hc6a6a1 = np.asarray(
    FD001_X_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float32,
)

y_dev_628hc6a6a1 = np.asarray(
    FD001_Y_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float32,
).reshape(-1)

if x_fit_628hc6a6a1.shape != (14459, 30, 17):
    raise RuntimeError(
        f"Unexpected FIT X shape: "
        f"{x_fit_628hc6a6a1.shape}"
    )

if y_fit_628hc6a6a1.shape != (14459,):
    raise RuntimeError(
        f"Unexpected FIT y shape: "
        f"{y_fit_628hc6a6a1.shape}"
    )

if x_dev_628hc6a6a1.shape != (3272, 30, 17):
    raise RuntimeError(
        f"Unexpected DEV X shape: "
        f"{x_dev_628hc6a6a1.shape}"
    )

if y_dev_628hc6a6a1.shape != (3272,):
    raise RuntimeError(
        f"Unexpected DEV y shape: "
        f"{y_dev_628hc6a6a1.shape}"
    )

if not all([
    np.isfinite(x_fit_628hc6a6a1).all(),
    np.isfinite(y_fit_628hc6a6a1).all(),
    np.isfinite(x_dev_628hc6a6a1).all(),
    np.isfinite(y_dev_628hc6a6a1).all(),
]):
    raise RuntimeError(
        "Non-finite FIT/DEV data detected."
    )

# ------------------------------------------------------------------
# 3. Reproducibility lock
# ------------------------------------------------------------------
seed_628hc6a6a1 = int(
    contract_628hc6a6a1["seed"]
)

random.seed(seed_628hc6a6a1)
np.random.seed(seed_628hc6a6a1)
torch.manual_seed(seed_628hc6a6a1)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        seed_628hc6a6a1
    )

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Use the same device family as the existing FD002 model.
device_628hc6a6a1 = next(
    MODEL_FD002_628EB.parameters()
).device

print(
    "\nTraining device:",
    device_628hc6a6a1,
)

# ------------------------------------------------------------------
# 4. Fresh FD001 model, same frozen architecture
# ------------------------------------------------------------------
model_class_628hc6a6a1 = (
    FD002GRURegressor628EB
)

sig_628hc6a6a1 = inspect.signature(
    model_class_628hc6a6a1
)

candidate_kwargs_628hc6a6a1 = {
    "input_size": 17,
    "hidden_size": 64,
    "num_layers": 1,
    "dense_hidden_size": 32,
    "dropout": 0.2,
}

kwargs_628hc6a6a1 = {
    k: v
    for k, v
    in candidate_kwargs_628hc6a6a1.items()
    if k in sig_628hc6a6a1.parameters
}

MODEL_FD001_628HC6A6A1 = (
    model_class_628hc6a6a1(
        **kwargs_628hc6a6a1
    )
    .to(device_628hc6a6a1)
)

ref_shapes_628hc6a6a1 = {
    k: tuple(v.shape)
    for k, v
    in MODEL_FD002_628EB.state_dict().items()
}

new_shapes_628hc6a6a1 = {
    k: tuple(v.shape)
    for k, v
    in MODEL_FD001_628HC6A6A1.state_dict().items()
}

param_count_628hc6a6a1 = int(
    sum(
        p.numel()
        for p in MODEL_FD001_628HC6A6A1.parameters()
    )
)

if (
    new_shapes_628hc6a6a1
    != ref_shapes_628hc6a6a1
):
    raise RuntimeError(
        "FD001 model state structure differs "
        "from frozen FD002 architecture."
    )

if param_count_628hc6a6a1 != 18049:
    raise RuntimeError(
        f"Unexpected model parameter count: "
        f"{param_count_628hc6a6a1}"
    )

print("\n--- ARCHITECTURE LOCK ---")
print(
    "Class             :",
    model_class_628hc6a6a1.__name__,
)
print(
    "Constructor kwargs:",
    kwargs_628hc6a6a1,
)
print(
    "Parameters        :",
    param_count_628hc6a6a1,
)
print(
    "State shapes exact:",
    True,
)

# ------------------------------------------------------------------
# 5. Data loaders
# ------------------------------------------------------------------
fit_ds_628hc6a6a1 = TensorDataset(
    torch.from_numpy(
        x_fit_628hc6a6a1
    ),
    torch.from_numpy(
        y_fit_628hc6a6a1
    ),
)

dev_ds_628hc6a6a1 = TensorDataset(
    torch.from_numpy(
        x_dev_628hc6a6a1
    ),
    torch.from_numpy(
        y_dev_628hc6a6a1
    ),
)

shuffle_gen_628hc6a6a1 = (
    torch.Generator()
)

shuffle_gen_628hc6a6a1.manual_seed(
    seed_628hc6a6a1
)

fit_loader_628hc6a6a1 = DataLoader(
    fit_ds_628hc6a6a1,
    batch_size=128,
    shuffle=True,
    generator=shuffle_gen_628hc6a6a1,
    num_workers=0,
)

dev_loader_628hc6a6a1 = DataLoader(
    dev_ds_628hc6a6a1,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

criterion_628hc6a6a1 = (
    torch.nn.MSELoss()
)

optimizer_628hc6a6a1 = (
    torch.optim.Adam(
        MODEL_FD001_628HC6A6A1.parameters(),
        lr=0.001,
    )
)

# ------------------------------------------------------------------
# 6. Training
#    FIT = parameter updates
#    DEV = frozen early-stopping criterion only
# ------------------------------------------------------------------
best_dev_mse_628hc6a6a1 = np.inf
best_epoch_628hc6a6a1 = 0
best_state_628hc6a6a1 = None
stale_628hc6a6a1 = 0

history_628hc6a6a1 = []

start_time_628hc6a6a1 = (
    time.perf_counter()
)

for epoch_628hc6a6a1 in range(
    1,
    101,
):

    MODEL_FD001_628HC6A6A1.train()

    fit_sum_628hc6a6a1 = 0.0
    fit_n_628hc6a6a1 = 0

    for (
        xb_628hc6a6a1,
        yb_628hc6a6a1,
    ) in fit_loader_628hc6a6a1:

        xb_628hc6a6a1 = (
            xb_628hc6a6a1.to(
                device_628hc6a6a1
            )
        )

        yb_628hc6a6a1 = (
            yb_628hc6a6a1.to(
                device_628hc6a6a1
            )
        )

        optimizer_628hc6a6a1.zero_grad(
            set_to_none=True
        )

        pred_628hc6a6a1 = (
            MODEL_FD001_628HC6A6A1(
                xb_628hc6a6a1
            )
            .reshape(-1)
        )

        loss_628hc6a6a1 = (
            criterion_628hc6a6a1(
                pred_628hc6a6a1,
                yb_628hc6a6a1,
            )
        )

        loss_628hc6a6a1.backward()

        torch.nn.utils.clip_grad_norm_(
            MODEL_FD001_628HC6A6A1.parameters(),
            1.0,
        )

        optimizer_628hc6a6a1.step()

        n_628hc6a6a1 = int(
            yb_628hc6a6a1.numel()
        )

        fit_sum_628hc6a6a1 += (
            loss_628hc6a6a1.item()
            * n_628hc6a6a1
        )

        fit_n_628hc6a6a1 += (
            n_628hc6a6a1
        )

    fit_mse_628hc6a6a1 = (
        fit_sum_628hc6a6a1
        / fit_n_628hc6a6a1
    )

    # DEV validation only
    MODEL_FD001_628HC6A6A1.eval()

    dev_sum_628hc6a6a1 = 0.0
    dev_n_628hc6a6a1 = 0

    with torch.no_grad():

        for (
            xb_628hc6a6a1,
            yb_628hc6a6a1,
        ) in dev_loader_628hc6a6a1:

            xb_628hc6a6a1 = (
                xb_628hc6a6a1.to(
                    device_628hc6a6a1
                )
            )

            yb_628hc6a6a1 = (
                yb_628hc6a6a1.to(
                    device_628hc6a6a1
                )
            )

            pred_628hc6a6a1 = (
                MODEL_FD001_628HC6A6A1(
                    xb_628hc6a6a1
                )
                .reshape(-1)
            )

            loss_628hc6a6a1 = (
                criterion_628hc6a6a1(
                    pred_628hc6a6a1,
                    yb_628hc6a6a1,
                )
            )

            n_628hc6a6a1 = int(
                yb_628hc6a6a1.numel()
            )

            dev_sum_628hc6a6a1 += (
                loss_628hc6a6a1.item()
                * n_628hc6a6a1
            )

            dev_n_628hc6a6a1 += (
                n_628hc6a6a1
            )

    dev_mse_628hc6a6a1 = (
        dev_sum_628hc6a6a1
        / dev_n_628hc6a6a1
    )

    improved_628hc6a6a1 = (
        dev_mse_628hc6a6a1
        < best_dev_mse_628hc6a6a1
    )

    if improved_628hc6a6a1:

        best_dev_mse_628hc6a6a1 = float(
            dev_mse_628hc6a6a1
        )

        best_epoch_628hc6a6a1 = int(
            epoch_628hc6a6a1
        )

        best_state_628hc6a6a1 = (
            copy.deepcopy(
                MODEL_FD001_628HC6A6A1.state_dict()
            )
        )

        stale_628hc6a6a1 = 0

    else:

        stale_628hc6a6a1 += 1

    history_628hc6a6a1.append(
        {
            "epoch":
                epoch_628hc6a6a1,

            "fit_mse":
                fit_mse_628hc6a6a1,

            "dev_mse":
                dev_mse_628hc6a6a1,

            "improved":
                improved_628hc6a6a1,

            "stale_epochs":
                stale_628hc6a6a1,
        }
    )

    print(
        f"epoch {epoch_628hc6a6a1:03d} | "
        f"FIT MSE {fit_mse_628hc6a6a1:10.4f} | "
        f"DEV MSE {dev_mse_628hc6a6a1:10.4f} | "
        f"best {best_epoch_628hc6a6a1:03d}"
    )

    if stale_628hc6a6a1 >= 12:

        print(
            "Early stopping at epoch "
            f"{epoch_628hc6a6a1}."
        )

        break

runtime_628hc6a6a1 = (
    time.perf_counter()
    - start_time_628hc6a6a1
)

if best_state_628hc6a6a1 is None:
    raise RuntimeError(
        "No valid best model state "
        "was produced."
    )

# Restore best DEV checkpoint
MODEL_FD001_628HC6A6A1.load_state_dict(
    best_state_628hc6a6a1
)

MODEL_FD001_628HC6A6A1.eval()

FD001_TRAINING_HISTORY_628HC6A6A1 = (
    pd.DataFrame(
        history_628hc6a6a1
    )
)

# ------------------------------------------------------------------
# 7. Restored best-model DEV predictions
# ------------------------------------------------------------------
dev_pred_parts_628hc6a6a1 = []

with torch.no_grad():

    for start_628hc6a6a1 in range(
        0,
        len(x_dev_628hc6a6a1),
        256,
    ):

        stop_628hc6a6a1 = min(
            start_628hc6a6a1 + 256,
            len(x_dev_628hc6a6a1),
        )

        xb_628hc6a6a1 = (
            torch.from_numpy(
                x_dev_628hc6a6a1[
                    start_628hc6a6a1:
                    stop_628hc6a6a1
                ]
            )
            .to(device_628hc6a6a1)
        )

        dev_pred_parts_628hc6a6a1.append(
            MODEL_FD001_628HC6A6A1(
                xb_628hc6a6a1
            )
            .reshape(-1)
            .cpu()
            .numpy()
        )

FD001_DEV_PREDICTIONS_628HC6A6A1 = (
    np.concatenate(
        dev_pred_parts_628hc6a6a1
    )
    .astype(np.float64)
)

dev_truth_628hc6a6a1 = (
    y_dev_628hc6a6a1.astype(
        np.float64
    )
)

dev_error_628hc6a6a1 = (
    FD001_DEV_PREDICTIONS_628HC6A6A1
    - dev_truth_628hc6a6a1
)

dev_mae_628hc6a6a1 = float(
    np.mean(
        np.abs(
            dev_error_628hc6a6a1
        )
    )
)

dev_rmse_628hc6a6a1 = float(
    np.sqrt(
        np.mean(
            dev_error_628hc6a6a1 ** 2
        )
    )
)

# ------------------------------------------------------------------
# 8. Model-state SHA-256
# ------------------------------------------------------------------
sha_628hc6a6a1 = hashlib.sha256()

for (
    key_628hc6a6a1,
    value_628hc6a6a1,
) in MODEL_FD001_628HC6A6A1.state_dict().items():

    sha_628hc6a6a1.update(
        key_628hc6a6a1.encode(
            "utf-8"
        )
    )

    sha_628hc6a6a1.update(
        value_628hc6a6a1
        .detach()
        .cpu()
        .contiguous()
        .numpy()
        .tobytes()
    )

FD001_MODEL_STATE_SHA256_628HC6A6A1 = (
    sha_628hc6a6a1.hexdigest()
)

FD001_MODEL_EVIDENCE_628HC6A6A1 = {
    "dataset": "FD001",

    "architecture":
        "single_layer_GRU_regressor",

    "fit_windows":
        14459,

    "dev_windows":
        3272,

    "best_epoch":
        best_epoch_628hc6a6a1,

    "epochs_executed":
        len(
            FD001_TRAINING_HISTORY_628HC6A6A1
        ),

    "best_dev_mse":
        best_dev_mse_628hc6a6a1,

    "dev_window_mae":
        dev_mae_628hc6a6a1,

    "dev_window_rmse":
        dev_rmse_628hc6a6a1,

    "parameter_count":
        param_count_628hc6a6a1,

    "model_state_sha256":
        FD001_MODEL_STATE_SHA256_628HC6A6A1,

    "training_runtime_sec":
        runtime_628hc6a6a1,

    "test_features_read":
        False,

    "official_test_labels_read":
        False,

    "test_scored":
        False,

    "reliability_threshold_selected":
        False,

    "files_written":
        False,
}

# ------------------------------------------------------------------
# 9. Audit
# ------------------------------------------------------------------
audit_628hc6a6a1 = pd.DataFrame(
    [
        (
            "c6_a6_a0_passed",
            True,
        ),

        (
            "frozen_training_contract_exact",
            not bool(
                bad_contract_628hc6a6a1
            ),
        ),

        (
            "fit_shape_14459x30x17",
            x_fit_628hc6a6a1.shape
            == (14459, 30, 17),
        ),

        (
            "dev_shape_3272x30x17",
            x_dev_628hc6a6a1.shape
            == (3272, 30, 17),
        ),

        (
            "fd002_architecture_class_reused",
            model_class_628hc6a6a1
            is FD002GRURegressor628EB,
        ),

        (
            "state_structure_exact_vs_fd002",
            new_shapes_628hc6a6a1
            == ref_shapes_628hc6a6a1,
        ),

        (
            "parameter_count_18049",
            param_count_628hc6a6a1
            == 18049,
        ),

        (
            "best_epoch_valid",
            1
            <= best_epoch_628hc6a6a1
            <= len(
                FD001_TRAINING_HISTORY_628HC6A6A1
            ),
        ),

        (
            "training_history_finite",
            np.isfinite(
                FD001_TRAINING_HISTORY_628HC6A6A1[
                    [
                        "fit_mse",
                        "dev_mse",
                    ]
                ]
            )
            .all()
            .all(),
        ),

        (
            "dev_predictions_shape_3272",
            FD001_DEV_PREDICTIONS_628HC6A6A1.shape
            == (3272,),
        ),

        (
            "dev_predictions_finite",
            np.isfinite(
                FD001_DEV_PREDICTIONS_628HC6A6A1
            ).all(),
        ),

        (
            "test_features_not_read",
            True,
        ),

        (
            "official_test_labels_not_read",
            True,
        ),

        (
            "test_not_scored",
            True,
        ),

        (
            "reliability_threshold_not_selected",
            True,
        ),

        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A6_A1_AUDIT_628HC6A6A1 = (
    audit_628hc6a6a1
)

C6_A6_A1_PASSED_628HC6A6A1 = bool(
    audit_628hc6a6a1[
        "passed"
    ].all()
)

print("\n--- C6-A6-A1 AUDIT ---")
display(
    C6_A6_A1_AUDIT_628HC6A6A1
)

if not C6_A6_A1_PASSED_628HC6A6A1:

    failed_628hc6a6a1 = (
        audit_628hc6a6a1.loc[
            ~audit_628hc6a6a1[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A6-A1 failed: "
        + ", ".join(
            failed_628hc6a6a1
        )
    )

# ------------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 116)

print(
    "STEP 6.28H-C6-A6-A1 FINAL SUMMARY"
)

print("=" * 116)

print(
    "Architecture                    :",
    contract_628hc6a6a1[
        "architecture"
    ],
)

print(
    "FIT windows used for fitting    :",
    len(
        x_fit_628hc6a6a1
    ),
)

print(
    "DEV windows used for validation :",
    len(
        x_dev_628hc6a6a1
    ),
)

print(
    "Best epoch                      :",
    best_epoch_628hc6a6a1,
)

print(
    "Epochs executed                 :",
    len(
        FD001_TRAINING_HISTORY_628HC6A6A1
    ),
)

print(
    "Best DEV MSE                    :",
    best_dev_mse_628hc6a6a1,
)

print(
    "Restored DEV window MAE         :",
    dev_mae_628hc6a6a1,
)

print(
    "Restored DEV window RMSE        :",
    dev_rmse_628hc6a6a1,
)

print(
    "Training runtime (sec)          :",
    runtime_628hc6a6a1,
)

print(
    "Model-state SHA-256             :",
    FD001_MODEL_STATE_SHA256_628HC6A6A1,
)

print(
    "Official TEST features read     :",
    False,
)

print(
    "Official FD001 TEST labels read :",
    False,
)

print(
    "TEST scored                     :",
    False,
)

print(
    "Reliability threshold selected  :",
    False,
)

print(
    "Files written                   :",
    False,
)

print(
    "All C6-A6-A1 checks passed      :",
    C6_A6_A1_PASSED_628HC6A6A1,
)

print(
    "\nNext required step: C6-A7 — "
    "CLEAN-DEV reliability construction using "
    "50-pass MC dropout, FIT-based uncertainty calibration, "
    "and FD001 DEV-only fixed-threshold derivation."
)

print(
    "\n=== END STEP 6.28H-C6-A6-A1 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A0 — FD002 UNCERTAINTY/RISK CALIBRATION CONTRACT RECOVERY
# ======================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-A0 — FD002 UNCERTAINTY/RISK CALIBRATION CONTRACT RECOVERY")
print("=" * 118)

# -----------------------------------------------------------------------------
# 0. Prerequisite
# -----------------------------------------------------------------------------
if not bool(globals().get("C6_A6_A1_PASSED_628HC6A6A1", False)):
    raise RuntimeError(
        "C6-A6-A1 must pass before C6-A7-A0."
    )

# Make sure the newly trained FD001 model is left in evaluation mode.
MODEL_FD001_628HC6A6A1.eval()

# -----------------------------------------------------------------------------
# 1. Explicit high-priority historical FD002 reliability objects
# -----------------------------------------------------------------------------
priority_names_628hc6a7a0 = [
    "G1_REFERENCE_FD002_628FC",
    "G2_REFERENCE_FD002_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "FD002_PHASE_REFERENCES_628FB",
    "FD002_PHASE_REFERENCES_628FBR1",
    "phase_mc_reference_628hc0",
    "g2_reference_values_628hc0",
    "g2_reference_values_628hc0r2",
    "g2_reference_values_628hc1d",
    "g2_reference_values_628hc3a1",
    "g2_reference_weights_628hc0",
    "g2_reference_weights_628hc0r2",
    "g2_reference_weights_628hc1d",
    "g2_reference_weights_628hc3a1",
    "clean_uncertainty_risk_628hc3a0r1",
    "uncertainty_risk_matrix_628hc3a0r1",
]

priority_rows_628hc6a7a0 = []

for name_628hc6a7a0 in priority_names_628hc6a7a0:

    exists_628hc6a7a0 = (
        name_628hc6a7a0 in globals()
    )

    if not exists_628hc6a7a0:

        priority_rows_628hc6a7a0.append(
            {
                "name": name_628hc6a7a0,
                "exists": False,
                "type": None,
                "shape": None,
                "finite": None,
                "min": None,
                "median": None,
                "mean": None,
                "max": None,
            }
        )

        continue

    obj_628hc6a7a0 = globals()[
        name_628hc6a7a0
    ]

    obj_type_628hc6a7a0 = type(
        obj_628hc6a7a0
    ).__name__

    shape_628hc6a7a0 = getattr(
        obj_628hc6a7a0,
        "shape",
        None,
    )

    finite_628hc6a7a0 = None
    min_628hc6a7a0 = None
    median_628hc6a7a0 = None
    mean_628hc6a7a0 = None
    max_628hc6a7a0 = None

    if isinstance(
        obj_628hc6a7a0,
        np.ndarray,
    ):

        arr_628hc6a7a0 = np.asarray(
            obj_628hc6a7a0
        )

        if (
            arr_628hc6a7a0.size > 0
            and np.issubdtype(
                arr_628hc6a7a0.dtype,
                np.number,
            )
        ):

            finite_628hc6a7a0 = bool(
                np.isfinite(
                    arr_628hc6a7a0
                ).all()
            )

            min_628hc6a7a0 = float(
                np.min(
                    arr_628hc6a7a0
                )
            )

            median_628hc6a7a0 = float(
                np.median(
                    arr_628hc6a7a0
                )
            )

            mean_628hc6a7a0 = float(
                np.mean(
                    arr_628hc6a7a0
                )
            )

            max_628hc6a7a0 = float(
                np.max(
                    arr_628hc6a7a0
                )
            )

    priority_rows_628hc6a7a0.append(
        {
            "name": name_628hc6a7a0,
            "exists": True,
            "type": obj_type_628hc6a7a0,
            "shape": shape_628hc6a7a0,
            "finite": finite_628hc6a7a0,
            "min": min_628hc6a7a0,
            "median": median_628hc6a7a0,
            "mean": mean_628hc6a7a0,
            "max": max_628hc6a7a0,
        }
    )

FD002_RELIABILITY_PRIORITY_INVENTORY_628HC6A7A0 = (
    pd.DataFrame(
        priority_rows_628hc6a7a0
    )
)

print(
    "\n--- HIGH-PRIORITY FD002 RELIABILITY OBJECTS ---"
)

display(
    FD002_RELIABILITY_PRIORITY_INVENTORY_628HC6A7A0
)

# -----------------------------------------------------------------------------
# 2. Safe snapshot of notebook namespace
#    Important: snapshot first so globals() cannot change during iteration.
# -----------------------------------------------------------------------------
namespace_snapshot_628hc6a7a0 = list(
    globals().items()
)

reliability_tokens_628hc6a7a0 = (
    "uncert",
    "risk",
    "g1",
    "g2",
    "calib",
    "rank",
    "percent",
    "reference",
    "mc_dropout",
)

generation_tokens_628hc6a7a0 = (
    "628f",
    "628hc0",
    "628hc1",
    "628hc3",
)

# -----------------------------------------------------------------------------
# 3. Discover relevant callable helpers
# -----------------------------------------------------------------------------
helper_rows_628hc6a7a0 = []

for (
    name_628hc6a7a0,
    obj_628hc6a7a0,
) in namespace_snapshot_628hc6a7a0:

    lower_name_628hc6a7a0 = (
        str(name_628hc6a7a0).lower()
    )

    relevant_name_628hc6a7a0 = (
        any(
            token in lower_name_628hc6a7a0
            for token
            in reliability_tokens_628hc6a7a0
        )
        and any(
            token in lower_name_628hc6a7a0
            for token
            in generation_tokens_628hc6a7a0
        )
    )

    if not relevant_name_628hc6a7a0:
        continue

    if not callable(obj_628hc6a7a0):
        continue

    try:
        signature_628hc6a7a0 = str(
            inspect.signature(
                obj_628hc6a7a0
            )
        )
    except Exception:
        signature_628hc6a7a0 = (
            "<signature unavailable>"
        )

    helper_rows_628hc6a7a0.append(
        {
            "name": name_628hc6a7a0,
            "type": type(
                obj_628hc6a7a0
            ).__name__,
            "signature":
                signature_628hc6a7a0,
        }
    )

FD002_RELIABILITY_HELPER_INVENTORY_628HC6A7A0 = (
    pd.DataFrame(
        helper_rows_628hc6a7a0
    )
    .sort_values(
        "name",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- RELEVANT HISTORICAL RELIABILITY HELPERS ---"
)

if len(
    FD002_RELIABILITY_HELPER_INVENTORY_628HC6A7A0
) == 0:

    print(
        "No matching callable helpers found."
    )

else:

    display(
        FD002_RELIABILITY_HELPER_INVENTORY_628HC6A7A0
    )

# -----------------------------------------------------------------------------
# 4. Discover relevant non-callable calibration/reference objects
# -----------------------------------------------------------------------------
object_rows_628hc6a7a0 = []

for (
    name_628hc6a7a0,
    obj_628hc6a7a0,
) in namespace_snapshot_628hc6a7a0:

    lower_name_628hc6a7a0 = (
        str(name_628hc6a7a0).lower()
    )

    relevant_name_628hc6a7a0 = (
        any(
            token in lower_name_628hc6a7a0
            for token
            in reliability_tokens_628hc6a7a0
        )
        and any(
            token in lower_name_628hc6a7a0
            for token
            in generation_tokens_628hc6a7a0
        )
    )

    if not relevant_name_628hc6a7a0:
        continue

    if callable(obj_628hc6a7a0):
        continue

    obj_type_628hc6a7a0 = type(
        obj_628hc6a7a0
    ).__name__

    shape_628hc6a7a0 = getattr(
        obj_628hc6a7a0,
        "shape",
        None,
    )

    size_628hc6a7a0 = None

    if isinstance(
        obj_628hc6a7a0,
        np.ndarray,
    ):

        size_628hc6a7a0 = int(
            obj_628hc6a7a0.size
        )

    elif isinstance(
        obj_628hc6a7a0,
        (
            list,
            tuple,
            dict,
            set,
        ),
    ):

        size_628hc6a7a0 = len(
            obj_628hc6a7a0
        )

    object_rows_628hc6a7a0.append(
        {
            "name": name_628hc6a7a0,
            "type": obj_type_628hc6a7a0,
            "shape": shape_628hc6a7a0,
            "size_or_len":
                size_628hc6a7a0,
        }
    )

FD002_RELIABILITY_OBJECT_INVENTORY_628HC6A7A0 = (
    pd.DataFrame(
        object_rows_628hc6a7a0
    )
    .sort_values(
        "name",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- RELEVANT HISTORICAL RELIABILITY OBJECTS ---"
)

if len(
    FD002_RELIABILITY_OBJECT_INVENTORY_628HC6A7A0
) == 0:

    print(
        "No matching non-callable objects found."
    )

else:

    display(
        FD002_RELIABILITY_OBJECT_INVENTORY_628HC6A7A0
    )

# -----------------------------------------------------------------------------
# 5. Verify the exact historical clean uncertainty-risk object if available
# -----------------------------------------------------------------------------
clean_risk_exists_628hc6a7a0 = (
    "clean_uncertainty_risk_628hc3a0r1"
    in globals()
)

matrix_exists_628hc6a7a0 = (
    "uncertainty_risk_matrix_628hc3a0r1"
    in globals()
)

clean_risk_contract_ok_628hc6a7a0 = False

if clean_risk_exists_628hc6a7a0:

    clean_risk_628hc6a7a0 = np.asarray(
        clean_uncertainty_risk_628hc3a0r1,
        dtype=np.float64,
    )

    clean_risk_contract_ok_628hc6a7a0 = bool(
        clean_risk_628hc6a7a0.ndim == 1
        and clean_risk_628hc6a7a0.size == 8367
        and np.isfinite(
            clean_risk_628hc6a7a0
        ).all()
        and (
            clean_risk_628hc6a7a0 >= 0.0
        ).all()
        and (
            clean_risk_628hc6a7a0 <= 1.0
        ).all()
    )

print(
    "\n--- HISTORICAL CLEAN-RISK CONTRACT ---"
)

print(
    "clean uncertainty-risk object exists :",
    clean_risk_exists_628hc6a7a0,
)

print(
    "corrupted uncertainty-risk matrix exists:",
    matrix_exists_628hc6a7a0,
)

print(
    "clean uncertainty-risk contract valid:",
    clean_risk_contract_ok_628hc6a7a0,
)

# -----------------------------------------------------------------------------
# 6. MC-dropout helper lock
# -----------------------------------------------------------------------------
mc_helper_name_628hc6a7a0 = (
    "mc_dropout_predict_628hc0"
)

mc_helper_exists_628hc6a7a0 = (
    mc_helper_name_628hc6a7a0
    in globals()
    and callable(
        globals()[
            mc_helper_name_628hc6a7a0
        ]
    )
)

mc_signature_628hc6a7a0 = None

if mc_helper_exists_628hc6a7a0:

    mc_signature_628hc6a7a0 = str(
        inspect.signature(
            globals()[
                mc_helper_name_628hc6a7a0
            ]
        )
    )

print(
    "\n--- MC-DROPOUT TRANSFER LOCK ---"
)

print(
    "Helper exists :",
    mc_helper_exists_628hc6a7a0,
)

print(
    "Helper name   :",
    mc_helper_name_628hc6a7a0,
)

print(
    "Signature     :",
    mc_signature_628hc6a7a0,
)

print(
    "Required passes:",
    50,
)

print(
    "Required batch size:",
    512,
)

print(
    "Required seed:",
    42,
)

# -----------------------------------------------------------------------------
# 7. Audit
# -----------------------------------------------------------------------------
audit_628hc6a7a0 = pd.DataFrame(
    [
        (
            "c6_a6_a1_passed",
            bool(
                C6_A6_A1_PASSED_628HC6A6A1
            ),
        ),
        (
            "fd001_model_exists",
            "MODEL_FD001_628HC6A6A1"
            in globals(),
        ),
        (
            "fd001_model_parameter_count_18049",
            sum(
                p.numel()
                for p in
                MODEL_FD001_628HC6A6A1.parameters()
            )
            == 18049,
        ),
        (
            "mc_dropout_helper_present",
            mc_helper_exists_628hc6a7a0,
        ),
        (
            "historical_clean_uncertainty_risk_present",
            clean_risk_exists_628hc6a7a0,
        ),
        (
            "historical_clean_uncertainty_risk_valid",
            clean_risk_contract_ok_628hc6a7a0,
        ),
        (
            "historical_corrupted_uncertainty_risk_present",
            matrix_exists_628hc6a7a0,
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_model_execution_in_this_step",
            True,
        ),
        (
            "no_model_retraining_in_this_step",
            True,
        ),
        (
            "no_threshold_tuning_in_this_step",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A7_A0_AUDIT_628HC6A7A0 = (
    audit_628hc6a7a0
)

print(
    "\n--- C6-A7-A0 AUDIT ---"
)

display(
    C6_A7_A0_AUDIT_628HC6A7A0
)

# Do NOT fail merely because one historical convenience
# object is absent. The inventory is the purpose of this step.
core_checks_628hc6a7a0 = [
    "c6_a6_a1_passed",
    "fd001_model_exists",
    "fd001_model_parameter_count_18049",
    "mc_dropout_helper_present",
    "test_features_not_read",
    "official_test_labels_not_read",
    "test_not_scored",
    "no_model_execution_in_this_step",
    "no_model_retraining_in_this_step",
    "no_threshold_tuning_in_this_step",
    "no_files_written",
]

core_mask_628hc6a7a0 = (
    C6_A7_A0_AUDIT_628HC6A7A0[
        "check"
    ].isin(
        core_checks_628hc6a7a0
    )
)

C6_A7_A0_PASSED_628HC6A7A0 = bool(
    C6_A7_A0_AUDIT_628HC6A7A0.loc[
        core_mask_628hc6a7a0,
        "passed",
    ].all()
)

if not C6_A7_A0_PASSED_628HC6A7A0:

    failed_628hc6a7a0 = (
        C6_A7_A0_AUDIT_628HC6A7A0.loc[
            core_mask_628hc6a7a0
            & ~C6_A7_A0_AUDIT_628HC6A7A0[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A7-A0 core recovery failed: "
        + ", ".join(
            failed_628hc6a7a0
        )
    )

# -----------------------------------------------------------------------------
# 8. Final summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 118)
print("STEP 6.28H-C6-A7-A0 FINAL SUMMARY")
print("=" * 118)

print(
    "FD001 trained model ready             :",
    True,
)

print(
    "FD001 model parameter count           :",
    18049,
)

print(
    "Historical reliability helpers found  :",
    len(
        FD002_RELIABILITY_HELPER_INVENTORY_628HC6A7A0
    ),
)

print(
    "Historical reliability objects found  :",
    len(
        FD002_RELIABILITY_OBJECT_INVENTORY_628HC6A7A0
    ),
)

print(
    "Historical clean uncertainty risk found:",
    clean_risk_exists_628hc6a7a0,
)

print(
    "MC-dropout helper recovered            :",
    mc_helper_exists_628hc6a7a0,
)

print(
    "Model executed in this step            :",
    False,
)

print(
    "TEST features read                     :",
    False,
)

print(
    "Official FD001 TEST labels read        :",
    False,
)

print(
    "Threshold selected in this step        :",
    False,
)

print(
    "Files written                          :",
    False,
)

print(
    "Core C6-A7-A0 checks passed            :",
    C6_A7_A0_PASSED_628HC6A7A0,
)

print(
    "\nNext step: use the recovered FD002 "
    "reliability lineage to construct the exact "
    "FD001 FIT/DEV uncertainty calibration, then "
    "derive the frozen 0.80 operating threshold "
    "from CLEAN DEV only."
)

print(
    "\n=== END STEP 6.28H-C6-A7-A0 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A1 — EXACT FD002 RELIABILITY LINEAGE IDENTITY AUDIT
# ======================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-A1 — EXACT FD002 RELIABILITY LINEAGE IDENTITY AUDIT")
print("=" * 118)

if not bool(globals().get("C6_A7_A0_PASSED_628HC6A7A0", False)):
    raise RuntimeError("C6-A7-A0 must pass before C6-A7-A1.")

# -----------------------------------------------------------------------------
# 1. Required historical components
# -----------------------------------------------------------------------------
required_628hc6a7a1 = [
    "G1_REFERENCE_FD002_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "clean_uncertainty_risk_628hc3a0r1",
    "DEV_CLEAN_UNCERTAINTY_FD002_628FAR2",
    "phase_adjust_uncertainty_628hc0",
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
    "weighted_midrank_ecdf_628hc3a0r1",
]

presence_628hc6a7a1 = pd.DataFrame(
    {
        "name": required_628hc6a7a1,
        "exists": [name in globals() for name in required_628hc6a7a1],
    }
)

print("\n--- REQUIRED COMPONENTS ---")
display(presence_628hc6a7a1)

if not bool(presence_628hc6a7a1["exists"].all()):
    missing_628hc6a7a1 = presence_628hc6a7a1.loc[
        ~presence_628hc6a7a1["exists"], "name"
    ].tolist()

    raise RuntimeError(
        "Missing historical components: "
        + ", ".join(missing_628hc6a7a1)
    )

# -----------------------------------------------------------------------------
# 2. Critical historical function contracts
# -----------------------------------------------------------------------------
critical_functions_628hc6a7a1 = [
    phase_adjust_uncertainty_628hc0,
    fd002_phase_adjusted_uncertainty_628hc0r2,
    fd002_g1_uncertainty_risk_628hc0r2,
    apply_fd002_g2_weighted_midrank_628hc0r2,
    fd002_final_selective_risk_628hc0r2,
    weighted_midrank_ecdf_628hc3a0r1,
]

function_contract_rows_628hc6a7a1 = []

for fn_628hc6a7a1 in critical_functions_628hc6a7a1:
    code_obj_628hc6a7a1 = getattr(fn_628hc6a7a1, "__code__", None)

    deps_628hc6a7a1 = (
        tuple(code_obj_628hc6a7a1.co_names)
        if code_obj_628hc6a7a1 is not None
        else tuple()
    )

    function_contract_rows_628hc6a7a1.append(
        {
            "name": fn_628hc6a7a1.__name__,
            "signature": str(inspect.signature(fn_628hc6a7a1)),
            "global_dependencies": deps_628hc6a7a1,
        }
    )

FD002_RELIABILITY_FUNCTION_CONTRACT_628HC6A7A1 = pd.DataFrame(
    function_contract_rows_628hc6a7a1
)

print("\n--- CRITICAL FUNCTION CONTRACTS ---")
display(FD002_RELIABILITY_FUNCTION_CONTRACT_628HC6A7A1)

# -----------------------------------------------------------------------------
# 3. Historical clean uncertainty table
# -----------------------------------------------------------------------------
clean_table_628hc6a7a1 = DEV_CLEAN_UNCERTAINTY_FD002_628FAR2.copy()

print("\n--- HISTORICAL CLEAN UNCERTAINTY TABLE ---")
print("Shape  :", clean_table_628hc6a7a1.shape)
print("Columns:", list(clean_table_628hc6a7a1.columns))

numeric_cols_628hc6a7a1 = [
    c
    for c in clean_table_628hc6a7a1.columns
    if np.issubdtype(clean_table_628hc6a7a1[c].dtype, np.number)
]

column_summary_628hc6a7a1 = pd.DataFrame(
    [
        {
            "column": c,
            "min": float(clean_table_628hc6a7a1[c].min()),
            "median": float(clean_table_628hc6a7a1[c].median()),
            "mean": float(clean_table_628hc6a7a1[c].mean()),
            "max": float(clean_table_628hc6a7a1[c].max()),
        }
        for c in numeric_cols_628hc6a7a1
    ]
)

display(column_summary_628hc6a7a1)

# -----------------------------------------------------------------------------
# 4. Recover exact G1 input identity
# -----------------------------------------------------------------------------
g1_reference_628hc6a7a1 = np.asarray(
    G1_REFERENCE_FD002_628FC,
    dtype=np.float64,
)

g1_rows_628hc6a7a1 = []

for c_628hc6a7a1 in numeric_cols_628hc6a7a1:

    x_628hc6a7a1 = clean_table_628hc6a7a1[
        c_628hc6a7a1
    ].to_numpy(
        dtype=np.float64
    )

    if x_628hc6a7a1.shape != g1_reference_628hc6a7a1.shape:
        continue

    try:
        mapped_628hc6a7a1 = np.asarray(
            fd002_g1_uncertainty_risk_628hc0r2(
                x_628hc6a7a1
            ),
            dtype=np.float64,
        )

        max_abs_628hc6a7a1 = float(
            np.max(
                np.abs(
                    mapped_628hc6a7a1
                    - g1_reference_628hc6a7a1
                )
            )
        )

        exact_628hc6a7a1 = bool(
            np.array_equal(
                mapped_628hc6a7a1,
                g1_reference_628hc6a7a1,
            )
        )

        close_628hc6a7a1 = bool(
            np.allclose(
                mapped_628hc6a7a1,
                g1_reference_628hc6a7a1,
                rtol=0.0,
                atol=1e-14,
            )
        )

        error_628hc6a7a1 = None

    except Exception as exc_628hc6a7a1:

        max_abs_628hc6a7a1 = np.inf
        exact_628hc6a7a1 = False
        close_628hc6a7a1 = False

        error_628hc6a7a1 = (
            f"{type(exc_628hc6a7a1).__name__}: "
            f"{exc_628hc6a7a1}"
        )

    g1_rows_628hc6a7a1.append(
        {
            "candidate_input_column": c_628hc6a7a1,
            "exact_vs_G1_REFERENCE": exact_628hc6a7a1,
            "allclose_1e14": close_628hc6a7a1,
            "max_abs_error": max_abs_628hc6a7a1,
            "error": error_628hc6a7a1,
        }
    )

FD002_G1_INPUT_IDENTITY_628HC6A7A1 = (
    pd.DataFrame(
        g1_rows_628hc6a7a1
    )
    .sort_values(
        "max_abs_error",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

print("\n--- G1 INPUT IDENTITY ---")
display(FD002_G1_INPUT_IDENTITY_628HC6A7A1)

# -----------------------------------------------------------------------------
# 5. Recover exact final-risk mapping
# -----------------------------------------------------------------------------
clean_final_628hc6a7a1 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64,
)

mapping_candidates_628hc6a7a1 = [
    (
        "fd002_final_selective_risk_628hc0r2",
        fd002_final_selective_risk_628hc0r2,
    ),
    (
        "apply_fd002_g2_weighted_midrank_628hc0r2",
        apply_fd002_g2_weighted_midrank_628hc0r2,
    ),
    (
        "weighted_midrank_ecdf_628hc3a0r1",
        weighted_midrank_ecdf_628hc3a0r1,
    ),
]

mapping_rows_628hc6a7a1 = []

for (
    name_628hc6a7a1,
    fn_628hc6a7a1,
) in mapping_candidates_628hc6a7a1:

    try:
        candidate_628hc6a7a1 = np.asarray(
            fn_628hc6a7a1(
                g1_reference_628hc6a7a1
            ),
            dtype=np.float64,
        )

        max_abs_628hc6a7a1 = float(
            np.max(
                np.abs(
                    candidate_628hc6a7a1
                    - clean_final_628hc6a7a1
                )
            )
        )

        exact_628hc6a7a1 = bool(
            np.array_equal(
                candidate_628hc6a7a1,
                clean_final_628hc6a7a1,
            )
        )

        close_628hc6a7a1 = bool(
            np.allclose(
                candidate_628hc6a7a1,
                clean_final_628hc6a7a1,
                rtol=0.0,
                atol=1e-14,
            )
        )

        error_628hc6a7a1 = None

    except Exception as exc_628hc6a7a1:

        max_abs_628hc6a7a1 = np.inf
        exact_628hc6a7a1 = False
        close_628hc6a7a1 = False

        error_628hc6a7a1 = (
            f"{type(exc_628hc6a7a1).__name__}: "
            f"{exc_628hc6a7a1}"
        )

    mapping_rows_628hc6a7a1.append(
        {
            "candidate_mapping": name_628hc6a7a1,
            "exact_vs_clean_final": exact_628hc6a7a1,
            "allclose_1e14": close_628hc6a7a1,
            "max_abs_error": max_abs_628hc6a7a1,
            "error": error_628hc6a7a1,
        }
    )

FD002_FINAL_RISK_IDENTITY_628HC6A7A1 = (
    pd.DataFrame(
        mapping_rows_628hc6a7a1
    )
    .sort_values(
        "max_abs_error",
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

print("\n--- FINAL-RISK IDENTITY ---")
display(FD002_FINAL_RISK_IDENTITY_628HC6A7A1)

# -----------------------------------------------------------------------------
# 6. Historical phase-reference keys
# -----------------------------------------------------------------------------
phase_rows_628hc6a7a1 = []

for object_name_628hc6a7a1 in [
    "FD002_PHASE_REFERENCES_628FB",
    "FD002_PHASE_REFERENCES_628FBR1",
    "phase_mc_reference_628hc0",
]:

    obj_628hc6a7a1 = globals().get(
        object_name_628hc6a7a1
    )

    if not isinstance(
        obj_628hc6a7a1,
        dict,
    ):
        continue

    for (
        key_628hc6a7a1,
        value_628hc6a7a1,
    ) in obj_628hc6a7a1.items():

        phase_rows_628hc6a7a1.append(
            {
                "object": object_name_628hc6a7a1,
                "key": str(key_628hc6a7a1),
                "type": type(
                    value_628hc6a7a1
                ).__name__,
                "shape": getattr(
                    value_628hc6a7a1,
                    "shape",
                    None,
                ),
            }
        )

FD002_PHASE_REFERENCE_KEYS_628HC6A7A1 = pd.DataFrame(
    phase_rows_628hc6a7a1
)

print("\n--- PHASE REFERENCE KEYS ---")
display(FD002_PHASE_REFERENCE_KEYS_628HC6A7A1)

# -----------------------------------------------------------------------------
# 7. Audit
# -----------------------------------------------------------------------------
g1_identity_ok_628hc6a7a1 = bool(
    FD002_G1_INPUT_IDENTITY_628HC6A7A1[
        "allclose_1e14"
    ].any()
)

final_identity_ok_628hc6a7a1 = bool(
    FD002_FINAL_RISK_IDENTITY_628HC6A7A1[
        "allclose_1e14"
    ].any()
)

C6_A7_A1_AUDIT_628HC6A7A1 = pd.DataFrame(
    [
        (
            "c6_a7_a0_passed",
            True,
        ),
        (
            "required_components_present",
            True,
        ),
        (
            "g1_reference_shape_8367",
            g1_reference_628hc6a7a1.shape
            == (8367,),
        ),
        (
            "clean_final_shape_8367",
            clean_final_628hc6a7a1.shape
            == (8367,),
        ),
        (
            "g1_reference_finite",
            bool(
                np.isfinite(
                    g1_reference_628hc6a7a1
                ).all()
            ),
        ),
        (
            "clean_final_finite",
            bool(
                np.isfinite(
                    clean_final_628hc6a7a1
                ).all()
            ),
        ),
        (
            "g1_input_identity_recovered",
            g1_identity_ok_628hc6a7a1,
        ),
        (
            "final_mapping_identity_recovered",
            final_identity_ok_628hc6a7a1,
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_threshold_tuning",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print("\n--- C6-A7-A1 AUDIT ---")
display(C6_A7_A1_AUDIT_628HC6A7A1)

C6_A7_A1_PASSED_628HC6A7A1 = bool(
    C6_A7_A1_AUDIT_628HC6A7A1[
        "passed"
    ].all()
)

print("\n" + "=" * 118)
print("STEP 6.28H-C6-A7-A1 FINAL SUMMARY")
print("=" * 118)

print(
    "G1 input identity recovered        :",
    g1_identity_ok_628hc6a7a1,
)

print(
    "Final-risk mapping recovered       :",
    final_identity_ok_628hc6a7a1,
)

print(
    "TEST features read                 :",
    False,
)

print(
    "Official FD001 TEST labels read    :",
    False,
)

print(
    "Model executed                     :",
    False,
)

print(
    "Threshold tuned/selected           :",
    False,
)

print(
    "Files written                      :",
    False,
)

print(
    "All C6-A7-A1 checks passed         :",
    C6_A7_A1_PASSED_628HC6A7A1,
)

if C6_A7_A1_PASSED_628HC6A7A1:
    print(
        "\nNext required step: C6-A7-A2 — execute 50-pass MC dropout "
        "on FD001 FIT/DEV, build FD001-specific FIT calibration "
        "references with this recovered lineage, and derive the "
        "fixed 0.80 CLEAN-DEV threshold."
    )
else:
    print(
        "\nDo not run MC dropout yet. Use the identity tables above "
        "to resolve the remaining historical lineage mismatch first."
    )

print("\n=== END STEP 6.28H-C6-A7-A1 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A1-R1 — EXACT FD002 CLEAN RELIABILITY PIPELINE RECONSTRUCTION
# ======================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-A1-R1 — EXACT FD002 CLEAN RELIABILITY PIPELINE RECONSTRUCTION")
print("=" * 118)

# -----------------------------------------------------------------------------
# 0. Preconditions
# -----------------------------------------------------------------------------
if not bool(globals().get("C6_A7_A0_PASSED_628HC6A7A0", False)):
    raise RuntimeError("C6-A7-A0 must pass before C6-A7-A1-R1.")

required_names_628hc6a7a1r1 = [
    "DEV_CLEAN_UNCERTAINTY_FD002_628FAR2",
    "G1_REFERENCE_FD002_628FC",
    "clean_uncertainty_risk_628hc3a0r1",
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
]

missing_628hc6a7a1r1 = [
    name
    for name in required_names_628hc6a7a1r1
    if name not in globals()
]

if missing_628hc6a7a1r1:
    raise RuntimeError(
        "Missing required historical objects/functions: "
        + ", ".join(missing_628hc6a7a1r1)
    )

# -----------------------------------------------------------------------------
# 1. Recover exact historical clean inputs
# -----------------------------------------------------------------------------
clean_table_628hc6a7a1r1 = DEV_CLEAN_UNCERTAINTY_FD002_628FAR2

required_columns_628hc6a7a1r1 = [
    "deterministic_prediction",
    "mc_std",
]

missing_columns_628hc6a7a1r1 = [
    c
    for c in required_columns_628hc6a7a1r1
    if c not in clean_table_628hc6a7a1r1.columns
]

if missing_columns_628hc6a7a1r1:
    raise RuntimeError(
        "Historical clean uncertainty table is missing columns: "
        + ", ".join(missing_columns_628hc6a7a1r1)
    )

det_628hc6a7a1r1 = clean_table_628hc6a7a1r1[
    "deterministic_prediction"
].to_numpy(dtype=np.float64)

mc_std_628hc6a7a1r1 = clean_table_628hc6a7a1r1[
    "mc_std"
].to_numpy(dtype=np.float64)

g1_reference_628hc6a7a1r1 = np.asarray(
    G1_REFERENCE_FD002_628FC,
    dtype=np.float64,
).reshape(-1)

clean_final_reference_628hc6a7a1r1 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64,
).reshape(-1)

expected_shape_628hc6a7a1r1 = (8367,)

for name_628hc6a7a1r1, arr_628hc6a7a1r1 in [
    ("deterministic_prediction", det_628hc6a7a1r1),
    ("mc_std", mc_std_628hc6a7a1r1),
    ("G1_REFERENCE_FD002_628FC", g1_reference_628hc6a7a1r1),
    ("clean_uncertainty_risk_628hc3a0r1", clean_final_reference_628hc6a7a1r1),
]:
    if arr_628hc6a7a1r1.shape != expected_shape_628hc6a7a1r1:
        raise RuntimeError(
            f"{name_628hc6a7a1r1} shape mismatch: "
            f"expected {expected_shape_628hc6a7a1r1}, "
            f"got {arr_628hc6a7a1r1.shape}"
        )

    if not np.isfinite(arr_628hc6a7a1r1).all():
        raise RuntimeError(
            f"{name_628hc6a7a1r1} contains non-finite values."
        )

# -----------------------------------------------------------------------------
# 2. Reconstruct historical reliability chain in the CORRECT order
#
# deterministic prediction + MC std
#          ↓
# phase-adjusted uncertainty
#          ↓
# G1 uncertainty risk
#          ↓
# corrected G2 / final selective risk
# -----------------------------------------------------------------------------
adjusted_uncertainty_628hc6a7a1r1 = np.asarray(
    fd002_phase_adjusted_uncertainty_628hc0r2(
        det_628hc6a7a1r1,
        mc_std_628hc6a7a1r1,
    ),
    dtype=np.float64,
).reshape(-1)

g1_reconstructed_628hc6a7a1r1 = np.asarray(
    fd002_g1_uncertainty_risk_628hc0r2(
        adjusted_uncertainty_628hc6a7a1r1
    ),
    dtype=np.float64,
).reshape(-1)

final_reconstructed_628hc6a7a1r1 = np.asarray(
    fd002_final_selective_risk_628hc0r2(
        g1_reconstructed_628hc6a7a1r1
    ),
    dtype=np.float64,
).reshape(-1)

final_direct_g2_628hc6a7a1r1 = np.asarray(
    apply_fd002_g2_weighted_midrank_628hc0r2(
        g1_reconstructed_628hc6a7a1r1
    ),
    dtype=np.float64,
).reshape(-1)

for name_628hc6a7a1r1, arr_628hc6a7a1r1 in [
    ("adjusted_uncertainty", adjusted_uncertainty_628hc6a7a1r1),
    ("g1_reconstructed", g1_reconstructed_628hc6a7a1r1),
    ("final_reconstructed", final_reconstructed_628hc6a7a1r1),
    ("final_direct_g2", final_direct_g2_628hc6a7a1r1),
]:
    if arr_628hc6a7a1r1.shape != expected_shape_628hc6a7a1r1:
        raise RuntimeError(
            f"{name_628hc6a7a1r1} shape mismatch: "
            f"expected {expected_shape_628hc6a7a1r1}, "
            f"got {arr_628hc6a7a1r1.shape}"
        )

    if not np.isfinite(arr_628hc6a7a1r1).all():
        raise RuntimeError(
            f"{name_628hc6a7a1r1} contains non-finite values."
        )

# -----------------------------------------------------------------------------
# 3. Exact / numerical identity checks
# -----------------------------------------------------------------------------
g1_absdiff_628hc6a7a1r1 = np.abs(
    g1_reconstructed_628hc6a7a1r1
    - g1_reference_628hc6a7a1r1
)

final_absdiff_628hc6a7a1r1 = np.abs(
    final_reconstructed_628hc6a7a1r1
    - clean_final_reference_628hc6a7a1r1
)

direct_final_absdiff_628hc6a7a1r1 = np.abs(
    final_direct_g2_628hc6a7a1r1
    - clean_final_reference_628hc6a7a1r1
)

g1_exact_628hc6a7a1r1 = bool(
    np.array_equal(
        g1_reconstructed_628hc6a7a1r1,
        g1_reference_628hc6a7a1r1,
    )
)

g1_close_628hc6a7a1r1 = bool(
    np.allclose(
        g1_reconstructed_628hc6a7a1r1,
        g1_reference_628hc6a7a1r1,
        rtol=0.0,
        atol=1e-14,
    )
)

final_exact_628hc6a7a1r1 = bool(
    np.array_equal(
        final_reconstructed_628hc6a7a1r1,
        clean_final_reference_628hc6a7a1r1,
    )
)

final_close_628hc6a7a1r1 = bool(
    np.allclose(
        final_reconstructed_628hc6a7a1r1,
        clean_final_reference_628hc6a7a1r1,
        rtol=0.0,
        atol=1e-14,
    )
)

direct_final_exact_628hc6a7a1r1 = bool(
    np.array_equal(
        final_direct_g2_628hc6a7a1r1,
        clean_final_reference_628hc6a7a1r1,
    )
)

direct_final_close_628hc6a7a1r1 = bool(
    np.allclose(
        final_direct_g2_628hc6a7a1r1,
        clean_final_reference_628hc6a7a1r1,
        rtol=0.0,
        atol=1e-14,
    )
)

# -----------------------------------------------------------------------------
# 4. Pipeline summary
# -----------------------------------------------------------------------------
print("\n--- RECONSTRUCTED PIPELINE SUMMARY ---")

pipeline_summary_628hc6a7a1r1 = pd.DataFrame(
    [
        {
            "stage": "historical deterministic prediction",
            "shape": det_628hc6a7a1r1.shape,
            "min": float(det_628hc6a7a1r1.min()),
            "median": float(np.median(det_628hc6a7a1r1)),
            "mean": float(det_628hc6a7a1r1.mean()),
            "max": float(det_628hc6a7a1r1.max()),
        },
        {
            "stage": "historical MC std",
            "shape": mc_std_628hc6a7a1r1.shape,
            "min": float(mc_std_628hc6a7a1r1.min()),
            "median": float(np.median(mc_std_628hc6a7a1r1)),
            "mean": float(mc_std_628hc6a7a1r1.mean()),
            "max": float(mc_std_628hc6a7a1r1.max()),
        },
        {
            "stage": "phase-adjusted uncertainty",
            "shape": adjusted_uncertainty_628hc6a7a1r1.shape,
            "min": float(adjusted_uncertainty_628hc6a7a1r1.min()),
            "median": float(np.median(adjusted_uncertainty_628hc6a7a1r1)),
            "mean": float(adjusted_uncertainty_628hc6a7a1r1.mean()),
            "max": float(adjusted_uncertainty_628hc6a7a1r1.max()),
        },
        {
            "stage": "G1 risk reconstructed",
            "shape": g1_reconstructed_628hc6a7a1r1.shape,
            "min": float(g1_reconstructed_628hc6a7a1r1.min()),
            "median": float(np.median(g1_reconstructed_628hc6a7a1r1)),
            "mean": float(g1_reconstructed_628hc6a7a1r1.mean()),
            "max": float(g1_reconstructed_628hc6a7a1r1.max()),
        },
        {
            "stage": "final risk reconstructed",
            "shape": final_reconstructed_628hc6a7a1r1.shape,
            "min": float(final_reconstructed_628hc6a7a1r1.min()),
            "median": float(np.median(final_reconstructed_628hc6a7a1r1)),
            "mean": float(final_reconstructed_628hc6a7a1r1.mean()),
            "max": float(final_reconstructed_628hc6a7a1r1.max()),
        },
    ]
)

display(pipeline_summary_628hc6a7a1r1)

# -----------------------------------------------------------------------------
# 5. Identity results
# -----------------------------------------------------------------------------
print("\n--- IDENTITY RESULTS ---")

identity_results_628hc6a7a1r1 = pd.DataFrame(
    [
        {
            "comparison":
                "reconstructed G1 vs G1_REFERENCE_FD002_628FC",
            "array_equal":
                g1_exact_628hc6a7a1r1,
            "allclose_atol_1e14":
                g1_close_628hc6a7a1r1,
            "max_abs_difference":
                float(g1_absdiff_628hc6a7a1r1.max()),
            "mean_abs_difference":
                float(g1_absdiff_628hc6a7a1r1.mean()),
        },
        {
            "comparison":
                "final function vs frozen clean uncertainty risk",
            "array_equal":
                final_exact_628hc6a7a1r1,
            "allclose_atol_1e14":
                final_close_628hc6a7a1r1,
            "max_abs_difference":
                float(final_absdiff_628hc6a7a1r1.max()),
            "mean_abs_difference":
                float(final_absdiff_628hc6a7a1r1.mean()),
        },
        {
            "comparison":
                "direct corrected G2 vs frozen clean uncertainty risk",
            "array_equal":
                direct_final_exact_628hc6a7a1r1,
            "allclose_atol_1e14":
                direct_final_close_628hc6a7a1r1,
            "max_abs_difference":
                float(direct_final_absdiff_628hc6a7a1r1.max()),
            "mean_abs_difference":
                float(direct_final_absdiff_628hc6a7a1r1.mean()),
        },
    ]
)

display(identity_results_628hc6a7a1r1)

# -----------------------------------------------------------------------------
# 6. Audit
# -----------------------------------------------------------------------------
C6_A7_A1_R1_AUDIT_628HC6A7A1R1 = pd.DataFrame(
    [
        ("c6_a7_a0_passed", True),

        (
            "historical_clean_table_shape_8367",
            clean_table_628hc6a7a1r1.shape[0] == 8367,
        ),

        (
            "historical_inputs_finite",
            bool(
                np.isfinite(det_628hc6a7a1r1).all()
                and np.isfinite(mc_std_628hc6a7a1r1).all()
            ),
        ),

        (
            "phase_adjusted_uncertainty_finite",
            bool(
                np.isfinite(
                    adjusted_uncertainty_628hc6a7a1r1
                ).all()
            ),
        ),

        (
            "g1_reconstructed_shape_8367",
            g1_reconstructed_628hc6a7a1r1.shape == (8367,),
        ),

        (
            "final_reconstructed_shape_8367",
            final_reconstructed_628hc6a7a1r1.shape == (8367,),
        ),

        (
            "g1_identity_allclose_1e14",
            g1_close_628hc6a7a1r1,
        ),

        (
            "final_identity_allclose_1e14",
            final_close_628hc6a7a1r1,
        ),

        (
            "direct_g2_identity_allclose_1e14",
            direct_final_close_628hc6a7a1r1,
        ),

        ("test_features_not_read", True),
        ("official_test_labels_not_read", True),
        ("test_not_scored", True),
        ("no_model_execution", True),
        ("no_model_retraining", True),
        ("no_threshold_tuning", True),
        ("no_files_written", True),
    ],
    columns=["check", "passed"],
)

print("\n--- C6-A7-A1-R1 AUDIT ---")
display(C6_A7_A1_R1_AUDIT_628HC6A7A1R1)

C6_A7_A1_R1_PASSED_628HC6A7A1R1 = bool(
    C6_A7_A1_R1_AUDIT_628HC6A7A1R1[
        "passed"
    ].all()
)

# -----------------------------------------------------------------------------
# 7. Final summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 118)
print("STEP 6.28H-C6-A7-A1-R1 FINAL SUMMARY")
print("=" * 118)

print(
    "Correct ordered lineage used                 : True"
)

print(
    "G1 identity exact                            :",
    g1_exact_628hc6a7a1r1,
)

print(
    "G1 identity within 1e-14                     :",
    g1_close_628hc6a7a1r1,
)

print(
    "Final-risk identity exact                    :",
    final_exact_628hc6a7a1r1,
)

print(
    "Final-risk identity within 1e-14             :",
    final_close_628hc6a7a1r1,
)

print(
    "Direct corrected-G2 identity exact           :",
    direct_final_exact_628hc6a7a1r1,
)

print(
    "Direct corrected-G2 identity within 1e-14    :",
    direct_final_close_628hc6a7a1r1,
)

print(
    "TEST features read                           : False"
)

print(
    "Official FD001 TEST labels read              : False"
)

print(
    "Model executed/retrained in this repair step : False"
)

print(
    "Threshold tuned/selected                     : False"
)

print(
    "Files written                                : False"
)

print(
    "All C6-A7-A1-R1 checks passed                :",
    C6_A7_A1_R1_PASSED_628HC6A7A1R1,
)

if C6_A7_A1_R1_PASSED_628HC6A7A1R1:
    print(
        "\nNext required step: C6-A7-A2 — run the frozen "
        "50-pass MC-dropout protocol on FD001 FIT and CLEAN DEV, "
        "construct FD001-specific FIT uncertainty calibration "
        "references, and derive the fixed 0.80 operating threshold "
        "from CLEAN DEV only."
    )
else:
    print(
        "\nSTOP before FD001 MC dropout. The ordered historical "
        "pipeline still does not reproduce the frozen FD002 clean "
        "reliability vector."
    )

print("\n=== END STEP 6.28H-C6-A7-A1-R1 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A1-R2 — CORRECTED FD002 RELIABILITY-LINEAGE IDENTITY LOCK
#
# Purpose:
#   Correct the semantic target used in A7-A1/R1.
#   G1_REFERENCE_FD002_628FC is the reference for PHASE-ADJUSTED uncertainty,
#   not the final G1-risk vector.
#
# Safety:
#   - historical FD002 objects only
#   - NO model execution
#   - NO retraining
#   - NO FD001 TEST features
#   - NO official FD001 TEST labels
#   - NO threshold selection/tuning
#   - NO files written
# ======================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-A1-R2 — CORRECTED FD002 RELIABILITY-LINEAGE IDENTITY LOCK")
print("=" * 118)

# ------------------------------------------------------------------
# 1. Required historical objects/functions
# ------------------------------------------------------------------
required_names_628hc6a7a1r2 = [
    "DEV_CLEAN_UNCERTAINTY_FD002_628FAR2",
    "G1_REFERENCE_FD002_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "clean_uncertainty_risk_628hc3a0r1",
    "phase_adjust_uncertainty_628hc0",
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
]

missing_628hc6a7a1r2 = [
    name
    for name in required_names_628hc6a7a1r2
    if name not in globals()
]

if missing_628hc6a7a1r2:
    raise RuntimeError(
        "Missing required historical objects/functions: "
        + ", ".join(missing_628hc6a7a1r2)
    )

# ------------------------------------------------------------------
# 2. Historical clean inputs
# ------------------------------------------------------------------
hist_628hc6a7a1r2 = DEV_CLEAN_UNCERTAINTY_FD002_628FAR2

required_cols_628hc6a7a1r2 = [
    "deterministic_prediction",
    "mc_std",
]

missing_cols_628hc6a7a1r2 = [
    c
    for c in required_cols_628hc6a7a1r2
    if c not in hist_628hc6a7a1r2.columns
]

if missing_cols_628hc6a7a1r2:
    raise RuntimeError(
        "Historical clean uncertainty table is missing columns: "
        + ", ".join(missing_cols_628hc6a7a1r2)
    )

det_628hc6a7a1r2 = np.asarray(
    hist_628hc6a7a1r2["deterministic_prediction"],
    dtype=np.float64,
).reshape(-1)

mcstd_628hc6a7a1r2 = np.asarray(
    hist_628hc6a7a1r2["mc_std"],
    dtype=np.float64,
).reshape(-1)

phase_ref_628hc6a7a1r2 = np.asarray(
    G1_REFERENCE_FD002_628FC,
    dtype=np.float64,
).reshape(-1)

g1_ref_628hc6a7a1r2 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

final_ref_628hc6a7a1r2 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64,
).reshape(-1)

expected_len_628hc6a7a1r2 = 8367

if not (
    det_628hc6a7a1r2.shape == (expected_len_628hc6a7a1r2,)
    and mcstd_628hc6a7a1r2.shape == (expected_len_628hc6a7a1r2,)
    and phase_ref_628hc6a7a1r2.shape == (expected_len_628hc6a7a1r2,)
    and g1_ref_628hc6a7a1r2.shape == (expected_len_628hc6a7a1r2,)
    and final_ref_628hc6a7a1r2.shape == (expected_len_628hc6a7a1r2,)
):
    raise RuntimeError("Unexpected historical FD002 clean-vector shape.")

# ------------------------------------------------------------------
# 3. Reconstruct phase-adjusted uncertainty using known helpers
# ------------------------------------------------------------------
phase_candidate_a_628hc6a7a1r2 = np.asarray(
    phase_adjust_uncertainty_628hc0(
        det_628hc6a7a1r2,
        mcstd_628hc6a7a1r2,
    ),
    dtype=np.float64,
).reshape(-1)

phase_candidate_b_628hc6a7a1r2 = np.asarray(
    fd002_phase_adjusted_uncertainty_628hc0r2(
        det_628hc6a7a1r2,
        mcstd_628hc6a7a1r2,
    ),
    dtype=np.float64,
).reshape(-1)

phase_a_close_628hc6a7a1r2 = bool(
    np.allclose(
        phase_candidate_a_628hc6a7a1r2,
        phase_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

phase_b_close_628hc6a7a1r2 = bool(
    np.allclose(
        phase_candidate_b_628hc6a7a1r2,
        phase_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

phase_a_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            phase_candidate_a_628hc6a7a1r2
            - phase_ref_628hc6a7a1r2
        )
    )
)

phase_b_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            phase_candidate_b_628hc6a7a1r2
            - phase_ref_628hc6a7a1r2
        )
    )
)

if phase_a_close_628hc6a7a1r2:
    phase_exact_628hc6a7a1r2 = phase_candidate_a_628hc6a7a1r2
    phase_helper_used_628hc6a7a1r2 = "phase_adjust_uncertainty_628hc0"

elif phase_b_close_628hc6a7a1r2:
    phase_exact_628hc6a7a1r2 = phase_candidate_b_628hc6a7a1r2
    phase_helper_used_628hc6a7a1r2 = "fd002_phase_adjusted_uncertainty_628hc0r2"

else:
    raise RuntimeError(
        "Neither historical phase-adjustment helper reproduces "
        "G1_REFERENCE_FD002_628FC within atol=1e-14."
    )

# ------------------------------------------------------------------
# 4. Correct ordered lineage
# ------------------------------------------------------------------
g1_risk_628hc6a7a1r2 = np.asarray(
    fd002_g1_uncertainty_risk_628hc0r2(
        phase_exact_628hc6a7a1r2
    ),
    dtype=np.float64,
).reshape(-1)

final_from_function_628hc6a7a1r2 = np.asarray(
    fd002_final_selective_risk_628hc0r2(
        g1_risk_628hc6a7a1r2
    ),
    dtype=np.float64,
).reshape(-1)

final_direct_g2_628hc6a7a1r2 = np.asarray(
    apply_fd002_g2_weighted_midrank_628hc0r2(
        g1_risk_628hc6a7a1r2
    ),
    dtype=np.float64,
).reshape(-1)

# ------------------------------------------------------------------
# 5. Correct semantic identity checks
# ------------------------------------------------------------------
phase_identity_close_628hc6a7a1r2 = bool(
    np.allclose(
        phase_exact_628hc6a7a1r2,
        phase_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

g1_identity_close_628hc6a7a1r2 = bool(
    np.allclose(
        g1_risk_628hc6a7a1r2,
        g1_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

final_function_close_628hc6a7a1r2 = bool(
    np.allclose(
        final_from_function_628hc6a7a1r2,
        final_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

final_direct_close_628hc6a7a1r2 = bool(
    np.allclose(
        final_direct_g2_628hc6a7a1r2,
        final_ref_628hc6a7a1r2,
        rtol=0.0,
        atol=1e-14,
    )
)

phase_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            phase_exact_628hc6a7a1r2
            - phase_ref_628hc6a7a1r2
        )
    )
)

g1_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            g1_risk_628hc6a7a1r2
            - g1_ref_628hc6a7a1r2
        )
    )
)

final_function_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            final_from_function_628hc6a7a1r2
            - final_ref_628hc6a7a1r2
        )
    )
)

final_direct_maxerr_628hc6a7a1r2 = float(
    np.max(
        np.abs(
            final_direct_g2_628hc6a7a1r2
            - final_ref_628hc6a7a1r2
        )
    )
)

print("\n--- CORRECTED SEMANTIC LINEAGE ---")
print("phase-adjusted uncertainty reference : G1_REFERENCE_FD002_628FC")
print("G1-risk reference                    : G2_REFERENCE_VALUES_FD002_628FCR2")
print("final-risk reference                 : clean_uncertainty_risk_628hc3a0r1")
print("phase helper used                    :", phase_helper_used_628hc6a7a1r2)

identity_table_628hc6a7a1r2 = pd.DataFrame(
    [
        {
            "comparison": "phase-adjusted uncertainty vs G1_REFERENCE",
            "within_1e14": phase_identity_close_628hc6a7a1r2,
            "max_abs_difference": phase_maxerr_628hc6a7a1r2,
        },
        {
            "comparison": "G1 risk vs G2_REFERENCE_VALUES",
            "within_1e14": g1_identity_close_628hc6a7a1r2,
            "max_abs_difference": g1_maxerr_628hc6a7a1r2,
        },
        {
            "comparison": "final function vs frozen clean final risk",
            "within_1e14": final_function_close_628hc6a7a1r2,
            "max_abs_difference": final_function_maxerr_628hc6a7a1r2,
        },
        {
            "comparison": "direct G2 mapping vs frozen clean final risk",
            "within_1e14": final_direct_close_628hc6a7a1r2,
            "max_abs_difference": final_direct_maxerr_628hc6a7a1r2,
        },
    ]
)

display(identity_table_628hc6a7a1r2)

print("\n--- PHASE HELPER DIAGNOSTIC ---")
print(
    "phase_adjust_uncertainty_628hc0 max error       :",
    phase_a_maxerr_628hc6a7a1r2,
)
print(
    "fd002_phase_adjusted_uncertainty_628hc0r2 error :",
    phase_b_maxerr_628hc6a7a1r2,
)

# ------------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------------
audit_628hc6a7a1r2 = pd.DataFrame(
    [
        ("historical_clean_table_shape_8367", hist_628hc6a7a1r2.shape[0] == 8367),
        ("phase_reference_semantics_corrected", True),
        ("phase_identity_within_1e14", phase_identity_close_628hc6a7a1r2),
        ("g1_identity_within_1e14", g1_identity_close_628hc6a7a1r2),
        ("final_function_identity_within_1e14", final_function_close_628hc6a7a1r2),
        ("direct_g2_identity_within_1e14", final_direct_close_628hc6a7a1r2),
        ("test_features_not_read", True),
        ("official_test_labels_not_read", True),
        ("test_not_scored", True),
        ("no_model_execution", True),
        ("no_model_retraining", True),
        ("no_threshold_tuning", True),
        ("no_files_written", True),
    ],
    columns=["check", "passed"],
)

print("\n--- C6-A7-A1-R2 AUDIT ---")
display(audit_628hc6a7a1r2)

all_checks_passed_628hc6a7a1r2 = bool(
    audit_628hc6a7a1r2["passed"].all()
)

print("\n" + "=" * 118)
print("STEP 6.28H-C6-A7-A1-R2 FINAL SUMMARY")
print("=" * 118)

print(
    "Phase-adjusted uncertainty identity within 1e-14 :",
    phase_identity_close_628hc6a7a1r2,
)
print(
    "G1-risk identity within 1e-14                    :",
    g1_identity_close_628hc6a7a1r2,
)
print(
    "Final-risk identity within 1e-14                 :",
    final_function_close_628hc6a7a1r2,
)
print(
    "Direct G2 final-risk identity within 1e-14       :",
    final_direct_close_628hc6a7a1r2,
)

print("TEST features read                                : False")
print("Official FD001 TEST labels read                   : False")
print("Model executed/retrained in this step             : False")
print("Threshold tuned/selected                          : False")
print("Files written                                     : False")
print(
    "All C6-A7-A1-R2 checks passed                     :",
    all_checks_passed_628hc6a7a1r2,
)

if not all_checks_passed_628hc6a7a1r2:
    raise RuntimeError(
        "C6-A7-A1-R2 lineage identity lock did not pass. "
        "Do not run FD001 MC dropout yet."
    )

print(
    "\nLineage lock passed. The historical FD002 reliability "
    "semantics are now identified correctly."
)

print(
    "Next required step: C6-A7-A2 — run frozen 50-pass MC dropout "
    "on FD001 FIT and CLEAN DEV, construct the transferred uncertainty "
    "calibration from FIT only, then derive the 0.80 operating threshold "
    "from CLEAN DEV only."
)

print("\n=== END STEP 6.28H-C6-A7-A1-R2 ===")

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-F0 — FORENSIC FD002 RELIABILITY-LINEAGE INSPECTION (READ-ONLY)
#
# Purpose:
#   Inspect the exact historical helper implementations and the reference objects they
#   depend on. This cell does NOT execute the model, MC dropout, threshold tuning,
#   TEST scoring, or file writes.
# ======================================================================================

import inspect
import hashlib
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-F0 — FORENSIC FD002 RELIABILITY-LINEAGE INSPECTION")
print("=" * 118)

# ------------------------------------------------------------------
# 1. Reference-object fingerprints
# ------------------------------------------------------------------
reference_names = [
    "G1_REFERENCE_FD002_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "clean_uncertainty_risk_628hc3a0r1",
    "FD002_PHASE_REFERENCES_628FB",
    "FD002_PHASE_REFERENCES_628FBR1",
    "phase_mc_reference_628hc0",
]

rows = []

for name in reference_names:
    obj = globals().get(name, None)

    if obj is None:
        rows.append(
            {
                "name": name,
                "exists": False,
                "type": None,
                "shape_or_keys": None,
                "sha256_if_array": None,
            }
        )
        continue

    if isinstance(obj, np.ndarray):
        arr = np.asarray(obj)
        sha = hashlib.sha256(
            np.ascontiguousarray(arr).tobytes(order="C")
        ).hexdigest()

        rows.append(
            {
                "name": name,
                "exists": True,
                "type": type(obj).__name__,
                "shape_or_keys": str(arr.shape),
                "sha256_if_array": sha,
            }
        )

    elif isinstance(obj, dict):
        rows.append(
            {
                "name": name,
                "exists": True,
                "type": type(obj).__name__,
                "shape_or_keys": str(list(obj.keys())),
                "sha256_if_array": None,
            }
        )

    else:
        rows.append(
            {
                "name": name,
                "exists": True,
                "type": type(obj).__name__,
                "shape_or_keys": getattr(obj, "shape", None),
                "sha256_if_array": None,
            }
        )

print("\n--- REFERENCE OBJECT FINGERPRINTS ---")
display(pd.DataFrame(rows))

# ------------------------------------------------------------------
# 2. Phase-reference dictionary contents
# ------------------------------------------------------------------
for dict_name in [
    "FD002_PHASE_REFERENCES_628FB",
    "FD002_PHASE_REFERENCES_628FBR1",
    "phase_mc_reference_628hc0",
]:
    obj = globals().get(dict_name, None)

    print(f"\n--- {dict_name} ---")

    if not isinstance(obj, dict):
        print("Not present as a dict.")
        continue

    detail_rows = []

    for key, value in obj.items():
        arr = np.asarray(value)

        detail_rows.append(
            {
                "key": key,
                "shape": arr.shape,
                "dtype": str(arr.dtype),
                "finite": bool(np.isfinite(arr).all())
                if np.issubdtype(arr.dtype, np.number)
                else None,
                "min": float(np.min(arr))
                if arr.size and np.issubdtype(arr.dtype, np.number)
                else None,
                "median": float(np.median(arr))
                if arr.size and np.issubdtype(arr.dtype, np.number)
                else None,
                "max": float(np.max(arr))
                if arr.size and np.issubdtype(arr.dtype, np.number)
                else None,
            }
        )

    display(pd.DataFrame(detail_rows))

# ------------------------------------------------------------------
# 3. Exact historical helper contracts + source
# ------------------------------------------------------------------
function_names = [
    "assign_phase_628hc0",
    "phase_adjust_uncertainty_628hc0",
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
]

for fn_name in function_names:
    fn = globals().get(fn_name, None)

    print("\n" + "-" * 118)
    print(fn_name)
    print("-" * 118)

    if fn is None:
        print("MISSING")
        continue

    print("type      :", type(fn).__name__)

    try:
        print("signature :", inspect.signature(fn))
    except Exception as exc:
        print("signature : unavailable ->", repr(exc))

    code_obj = getattr(fn, "__code__", None)

    if code_obj is not None:
        print("co_names  :", code_obj.co_names)
        print("defaults  :", getattr(fn, "__defaults__", None))
        print("kwdefaults:", getattr(fn, "__kwdefaults__", None))

    try:
        print("\nSOURCE:")
        print(inspect.getsource(fn))
    except Exception as exc:
        print("SOURCE unavailable ->", repr(exc))

# ------------------------------------------------------------------
# 4. Safety summary
# ------------------------------------------------------------------
print("\n" + "=" * 118)
print("FORENSIC INSPECTION COMPLETE")
print("=" * 118)
print("Model executed                 : False")
print("MC dropout executed            : False")
print("FD001 TEST features read       : False")
print("Official FD001 TEST labels read: False")
print("Threshold tuned/selected       : False")
print("Files written                  : False")

print(
    "\nDo not run MC dropout yet. "
    "Send this output back for exact lineage reconstruction."
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-F1 — EXACT FD002 CALIBRATION-CONSTRUCTION FORENSICS (READ-ONLY)
#
# Purpose:
#   Recover the exact historical construction semantics needed to transfer the
#   uncertainty-only reliability pipeline from FD002 to FD001.
#
# Safety:
#   - no model execution
#   - no MC dropout
#   - no TEST feature read
#   - no TEST label read
#   - no threshold selection
#   - no file writes
# ======================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C6-A7-F1 — EXACT FD002 CALIBRATION-CONSTRUCTION FORENSICS")
print("=" * 122)

# ------------------------------------------------------------------
# 1. Inspect exact historical calibration helpers
# ------------------------------------------------------------------
helper_names = [
    "apply_phase_adjustment_628fbr1",
    "apply_g1_right_ecdf_628fc",
    "apply_engine_balanced_g2_628fcr2",
    "apply_current_g2_628fcr1",
    "empirical_midrank_cdf_628hc0",
]

for name in helper_names:
    fn = globals().get(name, None)

    print("\n" + "-" * 122)
    print(name)
    print("-" * 122)

    if fn is None:
        print("MISSING")
        continue

    print("type      :", type(fn).__name__)

    try:
        print("signature :", inspect.signature(fn))
    except Exception as exc:
        print("signature : unavailable ->", repr(exc))

    code_obj = getattr(fn, "__code__", None)

    if code_obj is not None:
        print("co_names  :", code_obj.co_names)
        print("defaults  :", getattr(fn, "__defaults__", None))
        print("kwdefaults:", getattr(fn, "__kwdefaults__", None))

    try:
        print("\nSOURCE:")
        print(inspect.getsource(fn))
    except Exception as exc:
        print("SOURCE unavailable ->", repr(exc))

# ------------------------------------------------------------------
# 2. Inspect phase-edge globals without mutating notebook state
# ------------------------------------------------------------------
snapshot = list(globals().items())

phase_edge_rows = []

for name, obj in snapshot:
    lname = name.lower()

    if ("phase" not in lname) or ("edge" not in lname):
        continue

    try:
        arr = np.asarray(obj)

        if arr.ndim != 1 or arr.size == 0 or arr.size > 50:
            continue

        numeric = np.issubdtype(arr.dtype, np.number)

        phase_edge_rows.append(
            {
                "name": name,
                "type": type(obj).__name__,
                "shape": arr.shape,
                "dtype": str(arr.dtype),
                "values": arr.tolist() if numeric else repr(obj),
            }
        )
    except Exception:
        continue

print("\n--- PHASE-EDGE GLOBALS ---")

if phase_edge_rows:
    display(pd.DataFrame(phase_edge_rows))
else:
    print("No phase-edge globals found.")

# ------------------------------------------------------------------
# 3. Distribution-level lineage identities
#
# IMPORTANT:
# Calibration reference distributions do not need to preserve sample
# order. Therefore compare SORTED values, not original row order.
# ------------------------------------------------------------------
clean_table = DEV_CLEAN_UNCERTAINTY_FD002_628FAR2

det_hist = np.asarray(
    clean_table["deterministic_prediction"],
    dtype=np.float64,
).reshape(-1)

std_hist = np.asarray(
    clean_table["mc_std"],
    dtype=np.float64,
).reshape(-1)

phase_adjusted = np.asarray(
    fd002_phase_adjusted_uncertainty_628hc0r2(
        det_hist,
        std_hist,
    ),
    dtype=np.float64,
).reshape(-1)

g1_risk = np.asarray(
    fd002_g1_uncertainty_risk_628hc0r2(
        phase_adjusted
    ),
    dtype=np.float64,
).reshape(-1)

final_risk = np.asarray(
    fd002_final_selective_risk_628hc0r2(
        g1_risk
    ),
    dtype=np.float64,
).reshape(-1)

g1_reference = np.asarray(
    G1_REFERENCE_FD002_628FC,
    dtype=np.float64,
).reshape(-1)

g2_reference_values = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

clean_frozen = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64,
).reshape(-1)

phase_vs_g1_ref_sorted = bool(
    phase_adjusted.shape == g1_reference.shape
    and np.allclose(
        np.sort(phase_adjusted, kind="mergesort"),
        np.sort(g1_reference, kind="mergesort"),
        rtol=0.0,
        atol=1e-14,
    )
)

g1_vs_g2_ref_sorted = bool(
    g1_risk.shape == g2_reference_values.shape
    and np.allclose(
        np.sort(g1_risk, kind="mergesort"),
        np.sort(g2_reference_values, kind="mergesort"),
        rtol=0.0,
        atol=1e-14,
    )
)

final_vs_frozen = bool(
    final_risk.shape == clean_frozen.shape
    and np.allclose(
        final_risk,
        clean_frozen,
        rtol=0.0,
        atol=1e-14,
    )
)

phase_sorted_maxdiff = float(
    np.max(
        np.abs(
            np.sort(phase_adjusted, kind="mergesort")
            -
            np.sort(g1_reference, kind="mergesort")
        )
    )
)

g1_sorted_maxdiff = float(
    np.max(
        np.abs(
            np.sort(g1_risk, kind="mergesort")
            -
            np.sort(g2_reference_values, kind="mergesort")
        )
    )
)

final_maxdiff = float(
    np.max(
        np.abs(
            final_risk
            -
            clean_frozen
        )
    )
)

identity_table = pd.DataFrame(
    [
        {
            "comparison":
                "sorted phase-adjusted DEV vs G1 reference distribution",
            "allclose_atol_1e14":
                phase_vs_g1_ref_sorted,
            "max_abs_difference":
                phase_sorted_maxdiff,
        },
        {
            "comparison":
                "sorted G1-risk DEV vs G2 reference-value distribution",
            "allclose_atol_1e14":
                g1_vs_g2_ref_sorted,
            "max_abs_difference":
                g1_sorted_maxdiff,
        },
        {
            "comparison":
                "ordered final risk vs frozen clean uncertainty risk",
            "allclose_atol_1e14":
                final_vs_frozen,
            "max_abs_difference":
                final_maxdiff,
        },
    ]
)

print("\n--- DISTRIBUTION / FINAL-RISK IDENTITY AUDIT ---")
display(identity_table)

# ------------------------------------------------------------------
# 4. G2 weight diagnostics
# ------------------------------------------------------------------
g2_weights = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

print("\n--- G2 WEIGHT DIAGNOSTICS ---")
print("count       :", len(g2_weights))
print("finite      :", bool(np.isfinite(g2_weights).all()))
print("positive    :", bool((g2_weights > 0.0).all()))
print("sum         :", float(np.sum(g2_weights)))
print("min         :", float(np.min(g2_weights)))
print("median      :", float(np.median(g2_weights)))
print("mean        :", float(np.mean(g2_weights)))
print("max         :", float(np.max(g2_weights)))

# ------------------------------------------------------------------
# 5. Safety summary
# ------------------------------------------------------------------
print("\n" + "=" * 122)
print("FORENSIC CONSTRUCTION INSPECTION COMPLETE")
print("=" * 122)

print("Model executed                 : False")
print("MC dropout executed            : False")
print("FD001 TEST features read       : False")
print("Official FD001 TEST labels read: False")
print("Threshold tuned/selected       : False")
print("Files written                  : False")

print(
    "\nSend this output back. "
    "It will determine the exact FD001 calibration construction "
    "before any MC-dropout run."
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A2-A0 — FD001 MC EXECUTION OBJECT LOCK (READ-ONLY)
#
# Purpose:
#   Resolve the exact FD001 trained model and the exact FIT/DEV window tensors
#   before running the expensive 50-pass MC-dropout stage.
#
# Safety:
#   - no model forward pass
#   - no MC dropout
#   - no TEST feature read
#   - no TEST label read
#   - no threshold selection
#   - no file writes
# ======================================================================================

import hashlib
import numpy as np
import pandas as pd
import torch

print("=" * 122)
print("STEP 6.28H-C6-A7-A2-A0 — FD001 MC EXECUTION OBJECT LOCK")
print("=" * 122)

EXPECTED_FIT_SHAPE = (14459, 30, 17)
EXPECTED_DEV_SHAPE = (3272, 30, 17)

EXPECTED_FIT_SHA256 = (
    "4c12198c2f806812a4973f74c17f288606bf78cac8f7d8341022265436bae323"
)

EXPECTED_DEV_SHA256 = (
    "f2283cffacd23caf5bc2083b005f8b35848db03ae54b59021d8b55adf59b2565"
)


def array_sha256_c6a7a2a0(arr):
    a = np.ascontiguousarray(np.asarray(arr))
    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def model_fingerprint_c6a7a2a0(model):
    h = hashlib.sha256()

    state = model.state_dict()

    for key in state:
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            key.encode("utf-8")
        )

        h.update(
            np.ascontiguousarray(
                tensor
            ).tobytes(order="C")
        )

    return h.hexdigest()


snapshot_c6a7a2a0 = list(
    globals().items()
)

# ------------------------------------------------------------------
# 1. Resolve exact FIT / DEV window arrays using locked hashes
# ------------------------------------------------------------------

window_rows_c6a7a2a0 = []

for name, obj in snapshot_c6a7a2a0:

    if not isinstance(
        obj,
        np.ndarray,
    ):
        continue

    if obj.shape not in (
        EXPECTED_FIT_SHAPE,
        EXPECTED_DEV_SHAPE,
    ):
        continue

    sha = array_sha256_c6a7a2a0(
        obj
    )

    window_rows_c6a7a2a0.append(
        {
            "name": name,
            "shape": obj.shape,
            "sha256": sha,
            "matches_locked_fit": bool(
                obj.shape
                ==
                EXPECTED_FIT_SHAPE
                and
                sha
                ==
                EXPECTED_FIT_SHA256
            ),
            "matches_locked_dev": bool(
                obj.shape
                ==
                EXPECTED_DEV_SHAPE
                and
                sha
                ==
                EXPECTED_DEV_SHA256
            ),
        }
    )


window_table_c6a7a2a0 = pd.DataFrame(
    window_rows_c6a7a2a0
)

print(
    "\n--- WINDOW CANDIDATES ---"
)

if len(
    window_table_c6a7a2a0
) > 0:

    display(
        window_table_c6a7a2a0
    )

else:

    print(
        "No ndarray candidates with the required FIT/DEV shapes were found."
    )


fit_names_c6a7a2a0 = [
    row["name"]
    for row in window_rows_c6a7a2a0
    if row["matches_locked_fit"]
]

dev_names_c6a7a2a0 = [
    row["name"]
    for row in window_rows_c6a7a2a0
    if row["matches_locked_dev"]
]


fit_resolved_c6a7a2a0 = (
    len(
        fit_names_c6a7a2a0
    )
    >=
    1
)

dev_resolved_c6a7a2a0 = (
    len(
        dev_names_c6a7a2a0
    )
    >=
    1
)


if fit_resolved_c6a7a2a0:

    FD001_FIT_WINDOWS_LOCKED_628HC6A7A2A0 = (
        globals()[
            fit_names_c6a7a2a0[0]
        ]
    )


if dev_resolved_c6a7a2a0:

    FD001_DEV_WINDOWS_LOCKED_628HC6A7A2A0 = (
        globals()[
            dev_names_c6a7a2a0[0]
        ]
    )


# ------------------------------------------------------------------
# 2. Resolve trained FD001 model candidate
# ------------------------------------------------------------------

model_rows_c6a7a2a0 = []

for name, obj in snapshot_c6a7a2a0:

    if not isinstance(
        obj,
        torch.nn.Module,
    ):
        continue

    parameter_count = int(
        sum(
            p.numel()
            for p in obj.parameters()
        )
    )

    if parameter_count != 18049:
        continue

    model_rows_c6a7a2a0.append(
        {
            "name": name,
            "class_name":
                obj.__class__.__name__,
            "parameter_count":
                parameter_count,
            "training_mode":
                bool(
                    obj.training
                ),
            "fingerprint":
                model_fingerprint_c6a7a2a0(
                    obj
                ),
            "fd001_name_hint":
                bool(
                    (
                        "fd001"
                        in
                        name.lower()
                    )
                    or
                    (
                        "6a6a1"
                        in
                        name.lower()
                    )
                ),
        }
    )


model_table_c6a7a2a0 = pd.DataFrame(
    model_rows_c6a7a2a0
)

print(
    "\n--- 18,049-PARAMETER MODEL CANDIDATES ---"
)

if len(
    model_table_c6a7a2a0
) > 0:

    display(
        model_table_c6a7a2a0
    )

else:

    print(
        "No 18,049-parameter torch model candidates were found."
    )


preferred_model_rows_c6a7a2a0 = [
    row
    for row in model_rows_c6a7a2a0
    if row["fd001_name_hint"]
]


model_resolved_c6a7a2a0 = False

model_resolution_reason_c6a7a2a0 = (
    "UNRESOLVED"
)


if len(
    preferred_model_rows_c6a7a2a0
) == 1:

    selected_model_name_c6a7a2a0 = (
        preferred_model_rows_c6a7a2a0[0][
            "name"
        ]
    )

    model_resolved_c6a7a2a0 = True

    model_resolution_reason_c6a7a2a0 = (
        "unique FD001/name-stage hint"
    )


elif len(
    preferred_model_rows_c6a7a2a0
) > 1:

    preferred_fingerprints = {
        row["fingerprint"]
        for row
        in preferred_model_rows_c6a7a2a0
    }

    if len(
        preferred_fingerprints
    ) == 1:

        selected_model_name_c6a7a2a0 = (
            preferred_model_rows_c6a7a2a0[0][
                "name"
            ]
        )

        model_resolved_c6a7a2a0 = True

        model_resolution_reason_c6a7a2a0 = (
            "multiple FD001 aliases with identical model state"
        )


if model_resolved_c6a7a2a0:

    FD001_MODEL_LOCKED_628HC6A7A2A0 = (
        globals()[
            selected_model_name_c6a7a2a0
        ]
    )


# ------------------------------------------------------------------
# 3. MC helper availability — inspect only
# ------------------------------------------------------------------

mc_helper_c6a7a2a0 = globals().get(
    "mc_dropout_predict_628hc0",
    None,
)

mc_helper_present_c6a7a2a0 = callable(
    mc_helper_c6a7a2a0
)


# ------------------------------------------------------------------
# 4. Final execution-lock summary
# ------------------------------------------------------------------

print(
    "\n--- EXECUTION LOCK SUMMARY ---"
)

print(
    "FIT windows resolved by exact SHA-256 :",
    fit_resolved_c6a7a2a0,
)

print(
    "FIT aliases                            :",
    fit_names_c6a7a2a0,
)

print(
    "DEV windows resolved by exact SHA-256 :",
    dev_resolved_c6a7a2a0,
)

print(
    "DEV aliases                            :",
    dev_names_c6a7a2a0,
)

print(
    "FD001 model resolved                  :",
    model_resolved_c6a7a2a0,
)


if model_resolved_c6a7a2a0:

    print(
        "Selected model object                 :",
        selected_model_name_c6a7a2a0,
    )

    print(
        "Model resolution reason              :",
        model_resolution_reason_c6a7a2a0,
    )

    print(
        "Selected model fingerprint           :",
        model_fingerprint_c6a7a2a0(
            FD001_MODEL_LOCKED_628HC6A7A2A0
        ),
    )

else:

    print(
        "Selected model object                 : NONE"
    )

    print(
        "Action needed                         : "
        "inspect candidate table only; "
        "do not run MC dropout yet."
    )


print(
    "MC-dropout helper present             :",
    mc_helper_present_c6a7a2a0,
)


execution_lock_passed_c6a7a2a0 = bool(
    fit_resolved_c6a7a2a0
    and
    dev_resolved_c6a7a2a0
    and
    model_resolved_c6a7a2a0
    and
    mc_helper_present_c6a7a2a0
)


print(
    "Execution lock passed                 :",
    execution_lock_passed_c6a7a2a0,
)


print(
    "\n--- SAFETY ---"
)

print(
    "Model forward pass executed           : False"
)

print(
    "MC dropout executed                   : False"
)

print(
    "FD001 TEST features read              : False"
)

print(
    "Official FD001 TEST labels read       : False"
)

print(
    "Threshold tuned/selected              : False"
)

print(
    "Files written                         : False"
)


if execution_lock_passed_c6a7a2a0:

    print(
        "\nPASS — exact FD001 model/windows are locked. "
        "Next cell may run the 50-pass FIT+DEV MC-dropout stage."
    )

else:

    print(
        "\nSTOP — do not run MC dropout until this object lock is resolved."
    )


print(
    "\n=== END STEP 6.28H-C6-A7-A2-A0 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A7-A2-A1 — FD001 FIT+DEV DETERMINISTIC + 50-PASS MC-DROPOUT EXECUTION
#
# Purpose:
#   Execute the frozen FD001 model on the already-locked FIT/DEV windows only.
#   Produce deterministic predictions plus MC-dropout mean/std for later
#   FIT-based uncertainty calibration and CLEAN-DEV threshold derivation.
#
# Safety:
#   - uses only already-locked FD001 FIT/DEV windows
#   - does NOT read FD001 TEST features
#   - does NOT read official FD001 TEST labels
#   - does NOT score TEST
#   - does NOT select/tune a reliability threshold
#   - does NOT retrain the model
#   - does NOT write files
# ======================================================================================

import time
import hashlib
import numpy as np
import pandas as pd
import torch

print("=" * 122)
print("STEP 6.28H-C6-A7-A2-A1 — FD001 FIT+DEV DETERMINISTIC + 50-PASS MC-DROPOUT EXECUTION")
print("=" * 122)

# ------------------------------------------------------------------
# 0. Hard prerequisites from the immediately preceding lock cell
# ------------------------------------------------------------------

required_names_628hc6a7a2a1 = [
    "execution_lock_passed_c6a7a2a0",
    "FD001_FIT_WINDOWS_LOCKED_628HC6A7A2A0",
    "FD001_DEV_WINDOWS_LOCKED_628HC6A7A2A0",
    "FD001_MODEL_LOCKED_628HC6A7A2A0",
    "mc_dropout_predict_628hc0",
]

missing_names_628hc6a7a2a1 = [
    name
    for name in required_names_628hc6a7a2a1
    if name not in globals()
]

if missing_names_628hc6a7a2a1:
    raise RuntimeError(
        "Missing prerequisite objects: "
        + ", ".join(missing_names_628hc6a7a2a1)
    )

if not bool(execution_lock_passed_c6a7a2a0):
    raise RuntimeError(
        "C6-A7-A2-A0 execution lock did not pass."
    )

X_fit_628hc6a7a2a1 = np.asarray(
    FD001_FIT_WINDOWS_LOCKED_628HC6A7A2A0
)

X_dev_628hc6a7a2a1 = np.asarray(
    FD001_DEV_WINDOWS_LOCKED_628HC6A7A2A0
)

model_628hc6a7a2a1 = FD001_MODEL_LOCKED_628HC6A7A2A0

EXPECTED_FIT_SHAPE_628HC6A7A2A1 = (14459, 30, 17)
EXPECTED_DEV_SHAPE_628HC6A7A2A1 = (3272, 30, 17)

MC_PASSES_628HC6A7A2A1 = 50
MC_BATCH_SIZE_628HC6A7A2A1 = 512
MC_SEED_628HC6A7A2A1 = 42

if X_fit_628hc6a7a2a1.shape != EXPECTED_FIT_SHAPE_628HC6A7A2A1:
    raise RuntimeError(
        f"FIT window shape changed: {X_fit_628hc6a7a2a1.shape}"
    )

if X_dev_628hc6a7a2a1.shape != EXPECTED_DEV_SHAPE_628HC6A7A2A1:
    raise RuntimeError(
        f"DEV window shape changed: {X_dev_628hc6a7a2a1.shape}"
    )


# ------------------------------------------------------------------
# 1. Model-state fingerprint
# ------------------------------------------------------------------

def model_state_sha256_628hc6a7a2a1(model):

    h = hashlib.sha256()

    for key, tensor in model.state_dict().items():

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            key.encode("utf-8")
        )

        h.update(
            np.ascontiguousarray(
                arr
            ).tobytes(order="C")
        )

    return h.hexdigest()


model_hash_before_628hc6a7a2a1 = (
    model_state_sha256_628hc6a7a2a1(
        model_628hc6a7a2a1
    )
)

model_mode_before_628hc6a7a2a1 = bool(
    model_628hc6a7a2a1.training
)


# ------------------------------------------------------------------
# 2. Deterministic batched inference
# ------------------------------------------------------------------

def deterministic_predict_locked_628hc6a7a2a1(
    model,
    X,
    batch_size=512,
):

    X_local = np.asarray(
        X
    )

    try:
        device_local = next(
            model.parameters()
        ).device

    except StopIteration:
        device_local = torch.device(
            "cpu"
        )

    original_mode_local = bool(
        model.training
    )

    model.eval()

    pieces_local = []

    with torch.no_grad():

        for start_local in range(
            0,
            len(X_local),
            int(batch_size),
        ):

            stop_local = min(
                start_local
                +
                int(batch_size),
                len(X_local),
            )

            xb_local = torch.as_tensor(
                X_local[
                    start_local:stop_local
                ],
                dtype=torch.float32,
                device=device_local,
            )

            yb_local = model(
                xb_local
            )

            if isinstance(
                yb_local,
                (tuple, list),
            ):
                yb_local = (
                    yb_local[0]
                )

            yb_local = (
                yb_local
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            pieces_local.append(
                yb_local
            )

    if original_mode_local:
        model.train()
    else:
        model.eval()

    return np.concatenate(
        pieces_local
    ).astype(
        np.float64,
        copy=False,
    )


# ------------------------------------------------------------------
# 3. Robust parser for historical MC helper output
# ------------------------------------------------------------------

def to_numpy_628hc6a7a2a1(
    obj
):

    if torch.is_tensor(
        obj
    ):
        return (
            obj
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        obj
    )


def parse_mc_output_628hc6a7a2a1(
    result,
    expected_length,
    passes,
):

    # --------------------------------------------------------------
    # Dict output
    # --------------------------------------------------------------

    if isinstance(
        result,
        dict,
    ):

        mean_keys = (
            "mc_mean",
            "mean",
            "prediction_mean",
            "mean_prediction",
            "pred_mean",
        )

        std_keys = (
            "mc_std",
            "std",
            "prediction_std",
            "std_prediction",
            "pred_std",
        )

        mean_obj = None
        std_obj = None

        for key in mean_keys:

            if key in result:
                mean_obj = (
                    result[key]
                )
                break

        for key in std_keys:

            if key in result:
                std_obj = (
                    result[key]
                )
                break

        if (
            mean_obj is not None
            and
            std_obj is not None
        ):

            mean_arr = (
                to_numpy_628hc6a7a2a1(
                    mean_obj
                )
                .reshape(-1)
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            std_arr = (
                to_numpy_628hc6a7a2a1(
                    std_obj
                )
                .reshape(-1)
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            if (
                mean_arr.shape
                ==
                (expected_length,)
                and
                std_arr.shape
                ==
                (expected_length,)
            ):

                return (
                    mean_arr,
                    std_arr,
                    "dict_mean_std",
                )

        sample_keys = (
            "samples",
            "mc_samples",
            "predictions",
            "mc_predictions",
            "draws",
        )

        for key in sample_keys:

            if key not in result:
                continue

            sample_arr = (
                to_numpy_628hc6a7a2a1(
                    result[key]
                )
            )

            if sample_arr.ndim != 2:
                continue

            if sample_arr.shape == (
                passes,
                expected_length,
            ):

                samples = sample_arr

            elif sample_arr.shape == (
                expected_length,
                passes,
            ):

                samples = (
                    sample_arr.T
                )

            else:
                continue

            return (
                np.mean(
                    samples,
                    axis=0,
                    dtype=np.float64,
                ),
                np.std(
                    samples,
                    axis=0,
                    ddof=0,
                    dtype=np.float64,
                ),
                f"dict_samples:{key}",
            )

    # --------------------------------------------------------------
    # Tuple/list output
    # --------------------------------------------------------------

    if isinstance(
        result,
        (tuple, list),
    ):

        one_d_candidates = []
        sample_candidate = None

        for idx, item in enumerate(
            result
        ):

            try:
                arr = (
                    to_numpy_628hc6a7a2a1(
                        item
                    )
                )

            except Exception:
                continue

            if (
                arr.ndim == 1
                and
                arr.shape
                ==
                (expected_length,)
            ):

                one_d_candidates.append(
                    (
                        idx,
                        arr.astype(
                            np.float64,
                            copy=False,
                        ),
                    )
                )

            elif arr.ndim == 2:

                if arr.shape == (
                    passes,
                    expected_length,
                ):

                    sample_candidate = (
                        idx,
                        arr,
                    )

                elif arr.shape == (
                    expected_length,
                    passes,
                ):

                    sample_candidate = (
                        idx,
                        arr.T,
                    )

        if len(
            one_d_candidates
        ) >= 2:

            mean_arr = (
                one_d_candidates[0][1]
            )

            std_arr = (
                one_d_candidates[1][1]
            )

            return (
                mean_arr,
                std_arr,
                "tuple_or_list_1d_pair",
            )

        if sample_candidate is not None:

            samples = (
                sample_candidate[1]
            )

            return (
                np.mean(
                    samples,
                    axis=0,
                    dtype=np.float64,
                ),
                np.std(
                    samples,
                    axis=0,
                    ddof=0,
                    dtype=np.float64,
                ),
                "tuple_or_list_samples",
            )

    # --------------------------------------------------------------
    # Raw sample matrix
    # --------------------------------------------------------------

    try:
        arr_result = (
            to_numpy_628hc6a7a2a1(
                result
            )
        )

    except Exception:
        arr_result = None

    if (
        arr_result is not None
        and
        arr_result.ndim == 2
    ):

        if arr_result.shape == (
            passes,
            expected_length,
        ):

            samples = (
                arr_result
            )

        elif arr_result.shape == (
            expected_length,
            passes,
        ):

            samples = (
                arr_result.T
            )

        else:
            samples = None

        if samples is not None:

            return (
                np.mean(
                    samples,
                    axis=0,
                    dtype=np.float64,
                ),
                np.std(
                    samples,
                    axis=0,
                    ddof=0,
                    dtype=np.float64,
                ),
                "raw_sample_matrix",
            )

    raise RuntimeError(
        "Unsupported return contract from "
        "mc_dropout_predict_628hc0. "
        f"Observed type: {type(result).__name__}"
    )


# ------------------------------------------------------------------
# 4. Deterministic FIT + DEV inference
# ------------------------------------------------------------------

start_total_628hc6a7a2a1 = (
    time.perf_counter()
)

print(
    "\n--- DETERMINISTIC INFERENCE ---"
)

start_det_628hc6a7a2a1 = (
    time.perf_counter()
)

FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1 = (
    deterministic_predict_locked_628hc6a7a2a1(
        model_628hc6a7a2a1,
        X_fit_628hc6a7a2a1,
        batch_size=
        MC_BATCH_SIZE_628HC6A7A2A1,
    )
)

FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1 = (
    deterministic_predict_locked_628hc6a7a2a1(
        model_628hc6a7a2a1,
        X_dev_628hc6a7a2a1,
        batch_size=
        MC_BATCH_SIZE_628HC6A7A2A1,
    )
)

det_runtime_628hc6a7a2a1 = (
    time.perf_counter()
    -
    start_det_628hc6a7a2a1
)

print(
    "FIT deterministic shape :",
    FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1.shape,
)

print(
    "DEV deterministic shape :",
    FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1.shape,
)

print(
    "Deterministic runtime(s) :",
    det_runtime_628hc6a7a2a1,
)


# ------------------------------------------------------------------
# 5. Frozen 50-pass MC dropout — FIT
# ------------------------------------------------------------------

print(
    "\n--- 50-PASS MC DROPOUT: FIT ---"
)

start_fit_mc_628hc6a7a2a1 = (
    time.perf_counter()
)

fit_mc_raw_628hc6a7a2a1 = (
    mc_dropout_predict_628hc0(
        X_fit_628hc6a7a2a1,
        model_628hc6a7a2a1,
        passes=
        MC_PASSES_628HC6A7A2A1,
        batch_size=
        MC_BATCH_SIZE_628HC6A7A2A1,
        seed=
        MC_SEED_628HC6A7A2A1,
    )
)

(
    FD001_FIT_MC_MEAN_628HC6A7A2A1,
    FD001_FIT_MC_STD_628HC6A7A2A1,
    fit_mc_contract_628hc6a7a2a1,
) = parse_mc_output_628hc6a7a2a1(
    fit_mc_raw_628hc6a7a2a1,
    expected_length=
    EXPECTED_FIT_SHAPE_628HC6A7A2A1[0],
    passes=
    MC_PASSES_628HC6A7A2A1,
)

fit_mc_runtime_628hc6a7a2a1 = (
    time.perf_counter()
    -
    start_fit_mc_628hc6a7a2a1
)

print(
    "FIT MC return contract :",
    fit_mc_contract_628hc6a7a2a1,
)

print(
    "FIT MC mean/std shapes :",
    FD001_FIT_MC_MEAN_628HC6A7A2A1.shape,
    FD001_FIT_MC_STD_628HC6A7A2A1.shape,
)

print(
    "FIT MC runtime(s)      :",
    fit_mc_runtime_628hc6a7a2a1,
)


# ------------------------------------------------------------------
# 6. Frozen 50-pass MC dropout — DEV
# ------------------------------------------------------------------

print(
    "\n--- 50-PASS MC DROPOUT: DEV ---"
)

start_dev_mc_628hc6a7a2a1 = (
    time.perf_counter()
)

dev_mc_raw_628hc6a7a2a1 = (
    mc_dropout_predict_628hc0(
        X_dev_628hc6a7a2a1,
        model_628hc6a7a2a1,
        passes=
        MC_PASSES_628HC6A7A2A1,
        batch_size=
        MC_BATCH_SIZE_628HC6A7A2A1,
        seed=
        MC_SEED_628HC6A7A2A1,
    )
)

(
    FD001_DEV_MC_MEAN_628HC6A7A2A1,
    FD001_DEV_MC_STD_628HC6A7A2A1,
    dev_mc_contract_628hc6a7a2a1,
) = parse_mc_output_628hc6a7a2a1(
    dev_mc_raw_628hc6a7a2a1,
    expected_length=
    EXPECTED_DEV_SHAPE_628HC6A7A2A1[0],
    passes=
    MC_PASSES_628HC6A7A2A1,
)

dev_mc_runtime_628hc6a7a2a1 = (
    time.perf_counter()
    -
    start_dev_mc_628hc6a7a2a1
)

print(
    "DEV MC return contract :",
    dev_mc_contract_628hc6a7a2a1,
)

print(
    "DEV MC mean/std shapes :",
    FD001_DEV_MC_MEAN_628HC6A7A2A1.shape,
    FD001_DEV_MC_STD_628HC6A7A2A1.shape,
)

print(
    "DEV MC runtime(s)      :",
    dev_mc_runtime_628hc6a7a2a1,
)


# ------------------------------------------------------------------
# 7. Restore mode + verify weights unchanged
# ------------------------------------------------------------------

if model_mode_before_628hc6a7a2a1:
    model_628hc6a7a2a1.train()
else:
    model_628hc6a7a2a1.eval()

model_hash_after_628hc6a7a2a1 = (
    model_state_sha256_628hc6a7a2a1(
        model_628hc6a7a2a1
    )
)

model_state_unchanged_628hc6a7a2a1 = bool(
    model_hash_after_628hc6a7a2a1
    ==
    model_hash_before_628hc6a7a2a1
)


# ------------------------------------------------------------------
# 8. Audit
# ------------------------------------------------------------------

audit_rows_628hc6a7a2a1 = [
    (
        "execution_lock_passed",
        bool(
            execution_lock_passed_c6a7a2a0
        ),
    ),
    (
        "fit_deterministic_shape_14459",
        FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1.shape
        ==
        (14459,),
    ),
    (
        "dev_deterministic_shape_3272",
        FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1.shape
        ==
        (3272,),
    ),
    (
        "fit_mc_mean_shape_14459",
        FD001_FIT_MC_MEAN_628HC6A7A2A1.shape
        ==
        (14459,),
    ),
    (
        "fit_mc_std_shape_14459",
        FD001_FIT_MC_STD_628HC6A7A2A1.shape
        ==
        (14459,),
    ),
    (
        "dev_mc_mean_shape_3272",
        FD001_DEV_MC_MEAN_628HC6A7A2A1.shape
        ==
        (3272,),
    ),
    (
        "dev_mc_std_shape_3272",
        FD001_DEV_MC_STD_628HC6A7A2A1.shape
        ==
        (3272,),
    ),
    (
        "fit_deterministic_finite",
        bool(
            np.isfinite(
                FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "dev_deterministic_finite",
        bool(
            np.isfinite(
                FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "fit_mc_mean_finite",
        bool(
            np.isfinite(
                FD001_FIT_MC_MEAN_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "fit_mc_std_finite",
        bool(
            np.isfinite(
                FD001_FIT_MC_STD_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "dev_mc_mean_finite",
        bool(
            np.isfinite(
                FD001_DEV_MC_MEAN_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "dev_mc_std_finite",
        bool(
            np.isfinite(
                FD001_DEV_MC_STD_628HC6A7A2A1
            ).all()
        ),
    ),
    (
        "fit_mc_std_nonnegative",
        bool(
            np.all(
                FD001_FIT_MC_STD_628HC6A7A2A1
                >=
                0.0
            )
        ),
    ),
    (
        "dev_mc_std_nonnegative",
        bool(
            np.all(
                FD001_DEV_MC_STD_628HC6A7A2A1
                >=
                0.0
            )
        ),
    ),
    (
        "model_state_sha256_unchanged",
        model_state_unchanged_628hc6a7a2a1,
    ),
    (
        "test_features_not_read",
        True,
    ),
    (
        "official_test_labels_not_read",
        True,
    ),
    (
        "test_not_scored",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_selection",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

audit_628hc6a7a2a1 = pd.DataFrame(
    audit_rows_628hc6a7a2a1,
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C6-A7-A2-A1 AUDIT ---"
)

display(
    audit_628hc6a7a2a1
)

all_checks_passed_628hc6a7a2a1 = bool(
    audit_628hc6a7a2a1[
        "passed"
    ].all()
)


# ------------------------------------------------------------------
# 9. Compact output summary
# ------------------------------------------------------------------

def vector_summary_628hc6a7a2a1(
    name,
    values,
):

    arr = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    return {
        "name": name,
        "shape": arr.shape,
        "min": float(
            np.min(arr)
        ),
        "median": float(
            np.median(arr)
        ),
        "mean": float(
            np.mean(arr)
        ),
        "max": float(
            np.max(arr)
        ),
    }


summary_628hc6a7a2a1 = pd.DataFrame(
    [
        vector_summary_628hc6a7a2a1(
            "FIT deterministic",
            FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1,
        ),
        vector_summary_628hc6a7a2a1(
            "FIT MC mean",
            FD001_FIT_MC_MEAN_628HC6A7A2A1,
        ),
        vector_summary_628hc6a7a2a1(
            "FIT MC std",
            FD001_FIT_MC_STD_628HC6A7A2A1,
        ),
        vector_summary_628hc6a7a2a1(
            "DEV deterministic",
            FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1,
        ),
        vector_summary_628hc6a7a2a1(
            "DEV MC mean",
            FD001_DEV_MC_MEAN_628HC6A7A2A1,
        ),
        vector_summary_628hc6a7a2a1(
            "DEV MC std",
            FD001_DEV_MC_STD_628HC6A7A2A1,
        ),
    ]
)

print(
    "\n--- OUTPUT SUMMARY ---"
)

display(
    summary_628hc6a7a2a1
)


total_runtime_628hc6a7a2a1 = (
    time.perf_counter()
    -
    start_total_628hc6a7a2a1
)


# ------------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 122
)

print(
    "STEP 6.28H-C6-A7-A2-A1 FINAL SUMMARY"
)

print(
    "=" * 122
)

print(
    "FD001 FIT windows processed        :",
    len(
        FD001_FIT_MC_STD_628HC6A7A2A1
    ),
)

print(
    "FD001 DEV windows processed        :",
    len(
        FD001_DEV_MC_STD_628HC6A7A2A1
    ),
)

print(
    "MC passes                          :",
    MC_PASSES_628HC6A7A2A1,
)

print(
    "MC batch size                      :",
    MC_BATCH_SIZE_628HC6A7A2A1,
)

print(
    "MC seed                            :",
    MC_SEED_628HC6A7A2A1,
)

print(
    "FIT MC-STD median                  :",
    float(
        np.median(
            FD001_FIT_MC_STD_628HC6A7A2A1
        )
    ),
)

print(
    "DEV MC-STD median                  :",
    float(
        np.median(
            FD001_DEV_MC_STD_628HC6A7A2A1
        )
    ),
)

print(
    "Model-state SHA-256 unchanged      :",
    model_state_unchanged_628hc6a7a2a1,
)

print(
    "Official FD001 TEST features read  : False"
)

print(
    "Official FD001 TEST labels read    : False"
)

print(
    "TEST scored                        : False"
)

print(
    "Model retrained                    : False"
)

print(
    "Reliability threshold selected     : False"
)

print(
    "Files written                      : False"
)

print(
    "Total runtime (sec)                :",
    total_runtime_628hc6a7a2a1,
)

print(
    "All C6-A7-A2-A1 checks passed      :",
    all_checks_passed_628hc6a7a2a1,
)

if not all_checks_passed_628hc6a7a2a1:
    raise RuntimeError(
        "C6-A7-A2-A1 numerical/safety audit failed. "
        "Do not proceed to calibration."
    )

print(
    "\nNext required step: C6-A7-A2-A2 — "
    "construct FD001 FIT phase/uncertainty references, "
    "calibrate CLEAN DEV uncertainty-only risk, and derive "
    "the frozen nominal-0.80 DEV threshold."
)

print(
    "\n=== END STEP 6.28H-C6-A7-A2-A1 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A7-A2-A2 — FD001 CLEAN-DEV RELIABILITY CALIBRATION + 0.80 LOCK
# ==================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A7-A2-A2 — FD001 CLEAN-DEV RELIABILITY CALIBRATION + 0.80 LOCK")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Prerequisites
# ------------------------------------------------------------------

required_628hc6a7a2a2 = [
    "all_checks_passed_628hc6a7a2a1",
    "FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1",
    "FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1",
    "FD001_FIT_MC_STD_628HC6A7A2A1",
    "FD001_DEV_MC_STD_628HC6A7A2A1",
    "clean_uncertainty_risk_628hc3a0r1",
]

missing_628hc6a7a2a2 = [
    name
    for name in required_628hc6a7a2a2
    if name not in globals()
]

if missing_628hc6a7a2a2:
    raise RuntimeError(
        "Missing prerequisites: "
        + ", ".join(missing_628hc6a7a2a2)
    )

if not bool(all_checks_passed_628hc6a7a2a1):
    raise RuntimeError(
        "C6-A7-A2-A1 did not pass."
    )

fit_pred_628hc6a7a2a2 = np.asarray(
    FD001_FIT_DETERMINISTIC_PRED_628HC6A7A2A1,
    dtype=np.float64,
).reshape(-1)

dev_pred_628hc6a7a2a2 = np.asarray(
    FD001_DEV_DETERMINISTIC_PRED_628HC6A7A2A1,
    dtype=np.float64,
).reshape(-1)

fit_std_628hc6a7a2a2 = np.asarray(
    FD001_FIT_MC_STD_628HC6A7A2A1,
    dtype=np.float64,
).reshape(-1)

dev_std_628hc6a7a2a2 = np.asarray(
    FD001_DEV_MC_STD_628HC6A7A2A1,
    dtype=np.float64,
).reshape(-1)

if (
    fit_pred_628hc6a7a2a2.shape != (14459,)
    or fit_std_628hc6a7a2a2.shape != (14459,)
):
    raise RuntimeError(
        "Unexpected FIT prediction / MC-STD shape."
    )

if (
    dev_pred_628hc6a7a2a2.shape != (3272,)
    or dev_std_628hc6a7a2a2.shape != (3272,)
):
    raise RuntimeError(
        "Unexpected DEV prediction / MC-STD shape."
    )

if not all(
    np.isfinite(x).all()
    for x in (
        fit_pred_628hc6a7a2a2,
        dev_pred_628hc6a7a2a2,
        fit_std_628hc6a7a2a2,
        dev_std_628hc6a7a2a2,
    )
):
    raise RuntimeError(
        "Non-finite FIT/DEV prediction or MC-STD values."
    )


# ------------------------------------------------------------------
# 1. Recover exact historical FD002 threshold quantile rule
# ------------------------------------------------------------------

FD002_FROZEN_THRESHOLD_AUDIT_628HC6A7A2A2 = (
    0.8003345892330226
)

fd002_clean_risk_628hc6a7a2a2 = np.asarray(
    clean_uncertainty_risk_628hc3a0r1,
    dtype=np.float64,
).reshape(-1)

if fd002_clean_risk_628hc6a7a2a2.shape != (8367,):
    raise RuntimeError(
        "Historical FD002 clean-risk shape is not (8367,)."
    )

threshold_methods_628hc6a7a2a2 = (
    "linear",
    "lower",
    "higher",
    "midpoint",
    "nearest",
)

threshold_rule_rows_628hc6a7a2a2 = []

for method_628hc6a7a2a2 in threshold_methods_628hc6a7a2a2:

    candidate_628hc6a7a2a2 = float(
        np.quantile(
            fd002_clean_risk_628hc6a7a2a2,
            0.80,
            method=method_628hc6a7a2a2,
        )
    )

    threshold_rule_rows_628hc6a7a2a2.append(
        (
            method_628hc6a7a2a2,
            candidate_628hc6a7a2a2,
            abs(
                candidate_628hc6a7a2a2
                -
                FD002_FROZEN_THRESHOLD_AUDIT_628HC6A7A2A2
            ),
        )
    )

threshold_rule_table_628hc6a7a2a2 = pd.DataFrame(
    threshold_rule_rows_628hc6a7a2a2,
    columns=[
        "method",
        "candidate_threshold",
        "abs_error_vs_frozen_fd002",
    ],
).sort_values(
    "abs_error_vs_frozen_fd002"
).reset_index(
    drop=True
)

print(
    "\n--- HISTORICAL FD002 THRESHOLD-RULE IDENTITY ---"
)

display(
    threshold_rule_table_628hc6a7a2a2
)

threshold_method_628hc6a7a2a2 = str(
    threshold_rule_table_628hc6a7a2a2.loc[
        0,
        "method",
    ]
)

threshold_rule_error_628hc6a7a2a2 = float(
    threshold_rule_table_628hc6a7a2a2.loc[
        0,
        "abs_error_vs_frozen_fd002",
    ]
)

if threshold_rule_error_628hc6a7a2a2 > 1e-14:
    raise RuntimeError(
        "Historical FD002 threshold rule was not reproduced "
        "exactly. Stop before freezing the FD001 threshold."
    )


# ------------------------------------------------------------------
# 2. Resolve the existing DEV window endpoint metadata
# ------------------------------------------------------------------

dev_meta_candidates_628hc6a7a2a2 = []

for name_628hc6a7a2a2, obj_628hc6a7a2a2 in list(
    globals().items()
):

    if (
        isinstance(
            obj_628hc6a7a2a2,
            pd.DataFrame,
        )
        and len(
            obj_628hc6a7a2a2
        )
        == 3272
        and {
            "engine_id",
            "start_cycle",
            "end_cycle",
        }.issubset(
            obj_628hc6a7a2a2.columns
        )
    ):

        dev_meta_candidates_628hc6a7a2a2.append(
            (
                name_628hc6a7a2a2,
                obj_628hc6a7a2a2,
            )
        )

dev_named_candidates_628hc6a7a2a2 = [
    item
    for item in dev_meta_candidates_628hc6a7a2a2
    if "dev" in item[0].lower()
]

if dev_named_candidates_628hc6a7a2a2:
    dev_meta_candidates_628hc6a7a2a2 = (
        dev_named_candidates_628hc6a7a2a2
    )

if not dev_meta_candidates_628hc6a7a2a2:
    raise RuntimeError(
        "Could not locate the 3272-row C6-A5-R1 DEV "
        "endpoint metadata."
    )

(
    dev_meta_name_628hc6a7a2a2,
    dev_meta_628hc6a7a2a2,
) = dev_meta_candidates_628hc6a7a2a2[0]

dev_engine_ids_628hc6a7a2a2 = np.asarray(
    dev_meta_628hc6a7a2a2[
        "engine_id"
    ]
).reshape(-1)

if (
    len(
        dev_engine_ids_628hc6a7a2a2
    )
    != 3272
    or np.unique(
        dev_engine_ids_628hc6a7a2a2
    ).size
    != 20
):
    raise RuntimeError(
        "Resolved DEV endpoint metadata contract is invalid."
    )

for (
    alias_name_628hc6a7a2a2,
    alias_df_628hc6a7a2a2,
) in dev_meta_candidates_628hc6a7a2a2[1:]:

    alias_ids_628hc6a7a2a2 = np.asarray(
        alias_df_628hc6a7a2a2[
            "engine_id"
        ]
    ).reshape(-1)

    if not np.array_equal(
        dev_engine_ids_628hc6a7a2a2,
        alias_ids_628hc6a7a2a2,
    ):
        raise RuntimeError(
            "Conflicting DEV endpoint metadata: "
            f"{dev_meta_name_628hc6a7a2a2} vs "
            f"{alias_name_628hc6a7a2a2}."
        )


# ------------------------------------------------------------------
# 3. FIT-only phase references
# ------------------------------------------------------------------

FD001_PHASE_EDGES_628HC6A7A2A2 = np.quantile(
    fit_pred_628hc6a7a2a2,
    [
        0.20,
        0.40,
        0.60,
        0.80,
    ],
    method="linear",
).astype(
    np.float64
)

fit_phase_628hc6a7a2a2 = np.digitize(
    fit_pred_628hc6a7a2a2,
    FD001_PHASE_EDGES_628HC6A7A2A2,
    right=True,
)

dev_phase_628hc6a7a2a2 = np.digitize(
    dev_pred_628hc6a7a2a2,
    FD001_PHASE_EDGES_628HC6A7A2A2,
    right=True,
)

FD001_PHASE_REFERENCES_628HC6A7A2A2 = {
    phase: np.sort(
        fit_std_628hc6a7a2a2[
            fit_phase_628hc6a7a2a2
            ==
            phase
        ]
    ).astype(
        np.float64
    )
    for phase in range(
        5
    )
}

if any(
    len(
        FD001_PHASE_REFERENCES_628HC6A7A2A2[
            phase
        ]
    )
    == 0
    for phase in range(
        5
    )
):
    raise RuntimeError(
        "At least one FIT phase reference is empty."
    )

if sum(
    len(
        values
    )
    for values in FD001_PHASE_REFERENCES_628HC6A7A2A2.values()
) != 14459:
    raise RuntimeError(
        "FIT phase-reference total is not 14459."
    )


def fd001_phase_adjust_628hc6a7a2a2(
    prediction,
    mc_std,
):

    prediction = np.asarray(
        prediction,
        dtype=np.float64,
    ).reshape(-1)

    mc_std = np.asarray(
        mc_std,
        dtype=np.float64,
    ).reshape(-1)

    if prediction.shape != mc_std.shape:
        raise ValueError(
            "Prediction/MC-STD shape mismatch."
        )

    phase = np.digitize(
        prediction,
        FD001_PHASE_EDGES_628HC6A7A2A2,
        right=True,
    )

    result = np.empty(
        len(
            mc_std
        ),
        dtype=np.float64,
    )

    for p in range(
        5
    ):

        mask = (
            phase
            ==
            p
        )

        if not np.any(
            mask
        ):
            continue

        reference = (
            FD001_PHASE_REFERENCES_628HC6A7A2A2[
                p
            ]
        )

        values = (
            mc_std[
                mask
            ]
        )

        left = np.searchsorted(
            reference,
            values,
            side="left",
        )

        right = np.searchsorted(
            reference,
            values,
            side="right",
        )

        result[
            mask
        ] = (
            left
            +
            right
        ) / (
            2.0
            *
            len(
                reference
            )
        )

    return np.clip(
        result,
        0.0,
        1.0,
    )


FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2 = (
    fd001_phase_adjust_628hc6a7a2a2(
        dev_pred_628hc6a7a2a2,
        dev_std_628hc6a7a2a2,
    )
)


# ------------------------------------------------------------------
# 4. G1 — CLEAN-DEV right ECDF
# ------------------------------------------------------------------

FD001_G1_REFERENCE_628HC6A7A2A2 = np.sort(
    FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2
).astype(
    np.float64
)


def fd001_g1_risk_628hc6a7a2a2(
    values
):

    values = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    return (
        np.searchsorted(
            FD001_G1_REFERENCE_628HC6A7A2A2,
            values,
            side="right",
        )
        /
        float(
            len(
                FD001_G1_REFERENCE_628HC6A7A2A2
            )
        )
    )


FD001_DEV_G1_RISK_628HC6A7A2A2 = (
    fd001_g1_risk_628hc6a7a2a2(
        FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2
    )
)


# ------------------------------------------------------------------
# 5. G2 — engine-balanced CLEAN-DEV weighted midrank
# ------------------------------------------------------------------

(
    unique_engines_628hc6a7a2a2,
    engine_counts_628hc6a7a2a2,
) = np.unique(
    dev_engine_ids_628hc6a7a2a2,
    return_counts=True,
)

engine_count_map_628hc6a7a2a2 = {
    int(
        engine
    ): int(
        count
    )
    for engine, count in zip(
        unique_engines_628hc6a7a2a2,
        engine_counts_628hc6a7a2a2,
    )
}

FD001_G2_REFERENCE_VALUES_628HC6A7A2A2 = (
    FD001_DEV_G1_RISK_628HC6A7A2A2.copy()
)

FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2 = np.asarray(
    [
        1.0
        /
        (
            len(
                unique_engines_628hc6a7a2a2
            )
            *
            engine_count_map_628hc6a7a2a2[
                int(
                    engine
                )
            ]
        )
        for engine in dev_engine_ids_628hc6a7a2a2
    ],
    dtype=np.float64,
)

if not np.isclose(
    FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2.sum(),
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError(
        "G2 engine-balanced weights do not sum to 1."
    )

g2_order_628hc6a7a2a2 = np.argsort(
    FD001_G2_REFERENCE_VALUES_628HC6A7A2A2,
    kind="mergesort",
)

g2_values_sorted_628hc6a7a2a2 = (
    FD001_G2_REFERENCE_VALUES_628HC6A7A2A2[
        g2_order_628hc6a7a2a2
    ]
)

g2_weights_sorted_628hc6a7a2a2 = (
    FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2[
        g2_order_628hc6a7a2a2
    ]
)

g2_prefix_628hc6a7a2a2 = np.concatenate(
    [
        np.array(
            [
                0.0
            ],
            dtype=np.float64,
        ),
        np.cumsum(
            g2_weights_sorted_628hc6a7a2a2,
            dtype=np.float64,
        ),
    ]
)


def fd001_final_risk_628hc6a7a2a2(
    values
):

    values = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    left = np.searchsorted(
        g2_values_sorted_628hc6a7a2a2,
        values,
        side="left",
    )

    right = np.searchsorted(
        g2_values_sorted_628hc6a7a2a2,
        values,
        side="right",
    )

    less = (
        g2_prefix_628hc6a7a2a2[
            left
        ]
    )

    equal = (
        g2_prefix_628hc6a7a2a2[
            right
        ]
        -
        g2_prefix_628hc6a7a2a2[
            left
        ]
    )

    return np.clip(
        less
        +
        0.5
        *
        equal,
        0.0,
        1.0,
    )


FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2 = (
    fd001_final_risk_628hc6a7a2a2(
        FD001_DEV_G1_RISK_628HC6A7A2A2
    )
)


# ------------------------------------------------------------------
# 6. Freeze nominal-0.80 CLEAN-DEV threshold
# ------------------------------------------------------------------

FD001_NOMINAL_COVERAGE_628HC6A7A2A2 = (
    0.80
)

FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2 = float(
    np.quantile(
        FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2,
        FD001_NOMINAL_COVERAGE_628HC6A7A2A2,
        method=threshold_method_628hc6a7a2a2,
    )
)

FD001_DEV_ACCEPT_MASK_628HC6A7A2A2 = (
    FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2
    <=
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

FD001_DEV_ABSTAIN_MASK_628HC6A7A2A2 = (
    ~FD001_DEV_ACCEPT_MASK_628HC6A7A2A2
)

accepted_628hc6a7a2a2 = int(
    FD001_DEV_ACCEPT_MASK_628HC6A7A2A2.sum()
)

abstained_628hc6a7a2a2 = int(
    FD001_DEV_ABSTAIN_MASK_628HC6A7A2A2.sum()
)

coverage_628hc6a7a2a2 = float(
    FD001_DEV_ACCEPT_MASK_628HC6A7A2A2.mean()
)

ties_628hc6a7a2a2 = int(
    np.sum(
        FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2
        ==
        FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
    )
)


# ------------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------------

phase_summary_628hc6a7a2a2 = pd.DataFrame(
    [
        {
            "phase": phase,
            "fit_reference_n": len(
                FD001_PHASE_REFERENCES_628HC6A7A2A2[
                    phase
                ]
            ),
            "fit_mc_std_median": float(
                np.median(
                    FD001_PHASE_REFERENCES_628HC6A7A2A2[
                        phase
                    ]
                )
            ),
            "dev_windows": int(
                np.sum(
                    dev_phase_628hc6a7a2a2
                    ==
                    phase
                )
            ),
        }
        for phase in range(
            5
        )
    ]
)

risk_summary_628hc6a7a2a2 = pd.DataFrame(
    [
        {
            "stage": name,
            "min": float(
                np.min(
                    arr
                )
            ),
            "median": float(
                np.median(
                    arr
                )
            ),
            "mean": float(
                np.mean(
                    arr
                )
            ),
            "max": float(
                np.max(
                    arr
                )
            ),
        }
        for name, arr in [
            (
                "DEV phase-adjusted",
                FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2,
            ),
            (
                "DEV G1 risk",
                FD001_DEV_G1_RISK_628HC6A7A2A2,
            ),
            (
                "DEV final risk",
                FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2,
            ),
        ]
    ]
)

print(
    "\n--- DEV METADATA ---"
)

print(
    "Resolved endpoint metadata :",
    dev_meta_name_628hc6a7a2a2,
)

print(
    "DEV engines                :",
    len(
        unique_engines_628hc6a7a2a2
    ),
)

print(
    "\n--- FIT PHASE REFERENCES ---"
)

print(
    "Phase edges:",
    FD001_PHASE_EDGES_628HC6A7A2A2,
)

display(
    phase_summary_628hc6a7a2a2
)

print(
    "\n--- CLEAN-DEV RISK ---"
)

display(
    risk_summary_628hc6a7a2a2
)


# ------------------------------------------------------------------
# 8. Audit
# ------------------------------------------------------------------

audit_628hc6a7a2a2 = pd.DataFrame(
    [
        (
            "previous_step_passed",
            bool(
                all_checks_passed_628hc6a7a2a1
            ),
        ),
        (
            "fd002_threshold_rule_recovered",
            threshold_rule_error_628hc6a7a2a2
            <=
            1e-14,
        ),
        (
            "phase_edges_4_finite_increasing",
            FD001_PHASE_EDGES_628HC6A7A2A2.shape
            ==
            (4,)
            and
            np.isfinite(
                FD001_PHASE_EDGES_628HC6A7A2A2
            ).all()
            and
            np.all(
                np.diff(
                    FD001_PHASE_EDGES_628HC6A7A2A2
                )
                >
                0
            ),
        ),
        (
            "phase_reference_total_14459",
            sum(
                len(
                    values
                )
                for values in
                FD001_PHASE_REFERENCES_628HC6A7A2A2.values()
            )
            ==
            14459,
        ),
        (
            "dev_metadata_3272_rows_20_engines",
            len(
                dev_engine_ids_628hc6a7a2a2
            )
            ==
            3272
            and
            len(
                unique_engines_628hc6a7a2a2
            )
            ==
            20,
        ),
        (
            "dev_phase_adjusted_valid",
            FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2.shape
            ==
            (3272,)
            and
            np.isfinite(
                FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2
            ).all(),
        ),
        (
            "dev_g1_risk_valid",
            FD001_DEV_G1_RISK_628HC6A7A2A2.shape
            ==
            (3272,)
            and
            np.all(
                (
                    FD001_DEV_G1_RISK_628HC6A7A2A2
                    >=
                    0.0
                )
                &
                (
                    FD001_DEV_G1_RISK_628HC6A7A2A2
                    <=
                    1.0
                )
            ),
        ),
        (
            "g2_weights_valid",
            FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2.shape
            ==
            (3272,)
            and
            np.all(
                FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2
                >
                0.0
            )
            and
            np.isclose(
                FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2.sum(),
                1.0,
                rtol=0.0,
                atol=1e-14,
            ),
        ),
        (
            "final_dev_risk_valid",
            FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2.shape
            ==
            (3272,)
            and
            np.isfinite(
                FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2
            ).all()
            and
            np.all(
                (
                    FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2
                    >=
                    0.0
                )
                &
                (
                    FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2
                    <=
                    1.0
                )
            ),
        ),
        (
            "threshold_valid",
            np.isfinite(
                FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
            )
            and
            0.0
            <=
            FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
            <=
            1.0,
        ),
        (
            "decision_count_3272",
            accepted_628hc6a7a2a2
            +
            abstained_628hc6a7a2a2
            ==
            3272,
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C6-A7-A2-A2 AUDIT ---"
)

display(
    audit_628hc6a7a2a2
)

all_checks_passed_628hc6a7a2a2 = bool(
    audit_628hc6a7a2a2[
        "passed"
    ].all()
)


# ------------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A7-A2-A2 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "Reliability architecture      : uncertainty_only"
)

print(
    "FIT phase-reference windows   : 14459"
)

print(
    "CLEAN DEV calibration windows : 3272"
)

print(
    "DEV engines balanced in G2    :",
    len(
        unique_engines_628hc6a7a2a2
    ),
)

print(
    "Historical threshold rule     :",
    threshold_method_628hc6a7a2a2,
)

print(
    "Nominal coverage              : 0.80"
)

print(
    "Frozen FD001 threshold        :",
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2,
)

print(
    "DEV accepted / abstained      :",
    accepted_628hc6a7a2a2,
    "/",
    abstained_628hc6a7a2a2,
)

print(
    "Realized CLEAN-DEV coverage   :",
    coverage_628hc6a7a2a2,
)

print(
    "Threshold ties                :",
    ties_628hc6a7a2a2,
)

print(
    "FD001 TEST features read      : False"
)

print(
    "Official TEST labels read     : False"
)

print(
    "TEST scored                   : False"
)

print(
    "Model executed/retrained here : False"
)

print(
    "Files written                 : False"
)

print(
    "All checks passed             :",
    all_checks_passed_628hc6a7a2a2,
)

if not all_checks_passed_628hc6a7a2a2:
    raise RuntimeError(
        "C6-A7-A2-A2 audit failed. Do not proceed."
    )

print(
    "\nNext required step: C6-A8 — evaluate the frozen "
    "FD001 uncertainty-only policy on CLEAN DEV and controlled "
    "DEV sensor corruptions, with TEST still untouched."
)

print(
    "\n=== END STEP 6.28H-C6-A7-A2-A2 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A0 — FD001 DEV CORRUPTION-BRIDGE / PROTOCOL FORENSIC LOCK
# ==================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A8-A0 — FD001 DEV CORRUPTION-BRIDGE / PROTOCOL FORENSIC LOCK")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Gate on completed FD001 calibration
# ------------------------------------------------------------------

if "all_checks_passed_628hc6a7a2a2" not in globals():
    raise RuntimeError(
        "Missing all_checks_passed_628hc6a7a2a2. "
        "Run C6-A7-A2-A2 first."
    )

if not bool(all_checks_passed_628hc6a7a2a2):
    raise RuntimeError(
        "C6-A7-A2-A2 did not pass. Stop before C6-A8."
    )


# ------------------------------------------------------------------
# 1. Resolve FD001 DEV raw dataframe
# ------------------------------------------------------------------

fd001_dev_raw_candidates_628hc6a8a0 = []

for name_628hc6a8a0, obj_628hc6a8a0 in list(globals().items()):

    if not isinstance(obj_628hc6a8a0, pd.DataFrame):
        continue

    if obj_628hc6a8a0.shape[0] != 3852:
        continue

    columns_lower_628hc6a8a0 = {
        str(c).lower()
        for c in obj_628hc6a8a0.columns
    }

    has_unit_628hc6a8a0 = (
        "unit" in columns_lower_628hc6a8a0
        or
        "engine_id" in columns_lower_628hc6a8a0
    )

    has_cycle_628hc6a8a0 = (
        "cycle" in columns_lower_628hc6a8a0
    )

    if has_unit_628hc6a8a0 and has_cycle_628hc6a8a0:

        fd001_dev_raw_candidates_628hc6a8a0.append(
            (
                name_628hc6a8a0,
                obj_628hc6a8a0,
            )
        )

fd001_named_dev_raw_628hc6a8a0 = [
    item_628hc6a8a0
    for item_628hc6a8a0 in fd001_dev_raw_candidates_628hc6a8a0
    if (
        "fd001" in item_628hc6a8a0[0].lower()
        and
        "dev" in item_628hc6a8a0[0].lower()
        and
        "raw" in item_628hc6a8a0[0].lower()
    )
]

if fd001_named_dev_raw_628hc6a8a0:
    fd001_dev_raw_candidates_628hc6a8a0 = (
        fd001_named_dev_raw_628hc6a8a0
    )

if len(fd001_dev_raw_candidates_628hc6a8a0) == 0:
    raise RuntimeError(
        "No 3852-row FD001 DEV raw dataframe candidate was found."
    )

fd001_dev_raw_table_628hc6a8a0 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a0,
            "shape": tuple(obj_628hc6a8a0.shape),
            "columns": list(obj_628hc6a8a0.columns),
        }
        for name_628hc6a8a0, obj_628hc6a8a0
        in fd001_dev_raw_candidates_628hc6a8a0
    ]
)

print("\n--- FD001 DEV RAW CANDIDATES ---")
display(fd001_dev_raw_table_628hc6a8a0)


# ------------------------------------------------------------------
# 2. Recover retained-sensor and 17-feature contracts
# ------------------------------------------------------------------

sensor_list_candidates_628hc6a8a0 = []
feature_list_candidates_628hc6a8a0 = []

for name_628hc6a8a0, obj_628hc6a8a0 in list(globals().items()):

    if not isinstance(obj_628hc6a8a0, (list, tuple)):
        continue

    values_628hc6a8a0 = list(obj_628hc6a8a0)

    if (
        len(values_628hc6a8a0) == 15
        and
        all(
            isinstance(x_628hc6a8a0, str)
            and
            x_628hc6a8a0.startswith("sensor_")
            for x_628hc6a8a0 in values_628hc6a8a0
        )
    ):

        sensor_list_candidates_628hc6a8a0.append(
            (
                name_628hc6a8a0,
                values_628hc6a8a0,
            )
        )

    if (
        len(values_628hc6a8a0) == 17
        and
        all(
            isinstance(x_628hc6a8a0, str)
            for x_628hc6a8a0 in values_628hc6a8a0
        )
        and
        sum(
            x_628hc6a8a0.startswith("sensor_")
            for x_628hc6a8a0 in values_628hc6a8a0
        ) == 15
    ):

        feature_list_candidates_628hc6a8a0.append(
            (
                name_628hc6a8a0,
                values_628hc6a8a0,
            )
        )

sensor_contract_table_628hc6a8a0 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a0,
            "length": len(values_628hc6a8a0),
            "values": values_628hc6a8a0,
        }
        for name_628hc6a8a0, values_628hc6a8a0
        in sensor_list_candidates_628hc6a8a0
    ]
)

feature_contract_table_628hc6a8a0 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a0,
            "length": len(values_628hc6a8a0),
            "values": values_628hc6a8a0,
        }
        for name_628hc6a8a0, values_628hc6a8a0
        in feature_list_candidates_628hc6a8a0
    ]
)

print("\n--- 15-SENSOR CONTRACT CANDIDATES ---")
display(sensor_contract_table_628hc6a8a0)

print("\n--- 17-FEATURE CONTRACT CANDIDATES ---")
display(feature_contract_table_628hc6a8a0)


# ------------------------------------------------------------------
# 3. Locate historical 315-scenario corruption manifest
# ------------------------------------------------------------------

manifest_required_cols_628hc6a8a0 = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
}

manifest_candidates_628hc6a8a0 = []

for name_628hc6a8a0, obj_628hc6a8a0 in list(globals().items()):

    if not isinstance(obj_628hc6a8a0, pd.DataFrame):
        continue

    if len(obj_628hc6a8a0) != 315:
        continue

    if manifest_required_cols_628hc6a8a0.issubset(
        set(obj_628hc6a8a0.columns)
    ):

        manifest_candidates_628hc6a8a0.append(
            (
                name_628hc6a8a0,
                obj_628hc6a8a0,
            )
        )

manifest_table_628hc6a8a0 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a0,
            "shape": tuple(obj_628hc6a8a0.shape),
            "families": sorted(
                obj_628hc6a8a0[
                    "family"
                ].astype(str).unique().tolist()
            ),
            "severities": sorted(
                obj_628hc6a8a0[
                    "severity"
                ].astype(str).unique().tolist()
            ),
        }
        for name_628hc6a8a0, obj_628hc6a8a0
        in manifest_candidates_628hc6a8a0
    ]
)

print("\n--- HISTORICAL 315-SCENARIO MANIFEST CANDIDATES ---")
display(manifest_table_628hc6a8a0)


# ------------------------------------------------------------------
# 4. Inspect corruption helpers — DO NOT EXECUTE
# ------------------------------------------------------------------

helper_names_628hc6a8a0 = [
    "apply_scenario_628hc0",
    "normalize_corrupted_sensor_628ha",
    "normalize_sensor_628hc0",
    "build_windows_628hc0",
    "apply_fd002_dropout_628gc",
    "validate_sensor_628gc",
]

helper_rows_628hc6a8a0 = []

for helper_name_628hc6a8a0 in helper_names_628hc6a8a0:

    helper_obj_628hc6a8a0 = globals().get(
        helper_name_628hc6a8a0
    )

    if helper_obj_628hc6a8a0 is None:

        helper_rows_628hc6a8a0.append(
            {
                "name": helper_name_628hc6a8a0,
                "exists": False,
                "signature": None,
                "global_dependencies": None,
            }
        )

        continue

    try:

        signature_628hc6a8a0 = str(
            inspect.signature(
                helper_obj_628hc6a8a0
            )
        )

    except Exception:

        signature_628hc6a8a0 = (
            "<signature unavailable>"
        )

    try:

        dependencies_628hc6a8a0 = list(
            helper_obj_628hc6a8a0.__code__.co_names
        )

    except Exception:

        dependencies_628hc6a8a0 = None

    helper_rows_628hc6a8a0.append(
        {
            "name": helper_name_628hc6a8a0,
            "exists": True,
            "signature": signature_628hc6a8a0,
            "global_dependencies": dependencies_628hc6a8a0,
        }
    )

helper_table_628hc6a8a0 = pd.DataFrame(
    helper_rows_628hc6a8a0
)

print("\n--- HISTORICAL CORRUPTION-BRIDGE HELPERS ---")
display(helper_table_628hc6a8a0)


# ------------------------------------------------------------------
# 5. Print exact source for the scenario corruption function
# ------------------------------------------------------------------

print("\n--- SOURCE: apply_scenario_628hc0 ---")

if "apply_scenario_628hc0" in globals():

    try:

        print(
            inspect.getsource(
                apply_scenario_628hc0
            )
        )

    except Exception as exc_628hc6a8a0:

        print(
            "Source unavailable:",
            repr(exc_628hc6a8a0),
        )

else:

    print(
        "apply_scenario_628hc0 not present."
    )


# ------------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------------

audit_628hc6a8a0 = pd.DataFrame(
    [
        (
            "c6_a7_a2_a2_passed",
            bool(
                all_checks_passed_628hc6a7a2a2
            ),
        ),
        (
            "fd001_dev_raw_candidate_found",
            len(
                fd001_dev_raw_candidates_628hc6a8a0
            ) >= 1,
        ),
        (
            "retained_sensor_contract_found",
            len(
                sensor_list_candidates_628hc6a8a0
            ) >= 1,
        ),
        (
            "model_feature_contract_found",
            len(
                feature_list_candidates_628hc6a8a0
            ) >= 1,
        ),
        (
            "historical_315_manifest_found",
            len(
                manifest_candidates_628hc6a8a0
            ) >= 1,
        ),
        (
            "apply_scenario_helper_present",
            "apply_scenario_628hc0"
            in globals(),
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print("\n--- C6-A8-A0 AUDIT ---")
display(audit_628hc6a8a0)

all_checks_passed_628hc6a8a0 = bool(
    audit_628hc6a8a0[
        "passed"
    ].all()
)


# ------------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------------

print("\n" + "=" * 118)
print("STEP 6.28H-C6-A8-A0 FINAL SUMMARY")
print("=" * 118)

print(
    "FD001 DEV raw candidate(s)        :",
    len(
        fd001_dev_raw_candidates_628hc6a8a0
    ),
)

print(
    "15-sensor contract candidate(s)   :",
    len(
        sensor_list_candidates_628hc6a8a0
    ),
)

print(
    "17-feature contract candidate(s)  :",
    len(
        feature_list_candidates_628hc6a8a0
    ),
)

print(
    "315-scenario manifest candidate(s):",
    len(
        manifest_candidates_628hc6a8a0
    ),
)

print(
    "apply_scenario helper present     :",
    "apply_scenario_628hc0"
    in globals(),
)

print("FD001 TEST features read          : False")
print("Official FD001 TEST labels read   : False")
print("Model executed/retrained          : False")
print("Threshold modified                : False")
print("Files written                     : False")

print(
    "All C6-A8-A0 checks passed        :",
    all_checks_passed_628hc6a8a0,
)

print(
    "\nNext: build the exact FD001 DEV corruption protocol "
    "using only the objects identified above."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A0 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A1 — FD001 CORRUPTION-SEVERITY TRANSFER CONTRACT FORENSICS
# ==================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A8-A1 — FD001 CORRUPTION-SEVERITY TRANSFER CONTRACT FORENSICS")
print("=" * 118)

# ------------------------------------------------------------------
# 0. Gate
# ------------------------------------------------------------------

if "all_checks_passed_628hc6a8a0" not in globals():
    raise RuntimeError(
        "Missing all_checks_passed_628hc6a8a0. "
        "Run C6-A8-A0 first."
    )

if not bool(all_checks_passed_628hc6a8a0):
    raise RuntimeError(
        "C6-A8-A0 did not pass. Stop before C6-A8-A1."
    )


# ------------------------------------------------------------------
# 1. Exact FD001 transfer objects
# ------------------------------------------------------------------

required_fd001_names_628hc6a8a1 = [
    "FD001_FIT_RAW_628HC6A3R1",
    "FD001_DEV_RAW_628HC6A3R1",
    "FD001_RETAINED_SENSORS_628HC6A4",
    "FD001_MODEL_FEATURES_628HC6A4",
]

fd001_presence_628hc6a8a1 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a1,
            "exists": name_628hc6a8a1 in globals(),
            "type": (
                type(globals()[name_628hc6a8a1]).__name__
                if name_628hc6a8a1 in globals()
                else None
            ),
            "shape_or_len": (
                tuple(globals()[name_628hc6a8a1].shape)
                if (
                    name_628hc6a8a1 in globals()
                    and hasattr(globals()[name_628hc6a8a1], "shape")
                )
                else (
                    len(globals()[name_628hc6a8a1])
                    if (
                        name_628hc6a8a1 in globals()
                        and hasattr(globals()[name_628hc6a8a1], "__len__")
                    )
                    else None
                )
            ),
        }
        for name_628hc6a8a1 in required_fd001_names_628hc6a8a1
    ]
)

print("\n--- EXACT FD001 TRANSFER OBJECTS ---")
display(fd001_presence_628hc6a8a1)


# ------------------------------------------------------------------
# 2. Exact historical FD002 corruption-protocol objects
# ------------------------------------------------------------------

historical_names_628hc6a8a1 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_FIT_CORRUPTION_REFERENCE_628GB",
]

historical_presence_628hc6a8a1 = pd.DataFrame(
    [
        {
            "name": name_628hc6a8a1,
            "exists": name_628hc6a8a1 in globals(),
            "type": (
                type(globals()[name_628hc6a8a1]).__name__
                if name_628hc6a8a1 in globals()
                else None
            ),
            "shape_or_keys": (
                tuple(globals()[name_628hc6a8a1].shape)
                if (
                    name_628hc6a8a1 in globals()
                    and hasattr(globals()[name_628hc6a8a1], "shape")
                )
                else (
                    sorted(list(globals()[name_628hc6a8a1].keys()))
                    if (
                        name_628hc6a8a1 in globals()
                        and isinstance(globals()[name_628hc6a8a1], dict)
                    )
                    else None
                )
            ),
        }
        for name_628hc6a8a1 in historical_names_628hc6a8a1
    ]
)

print("\n--- EXACT HISTORICAL FD002 CORRUPTION OBJECTS ---")
display(historical_presence_628hc6a8a1)


# ------------------------------------------------------------------
# 3. Lock the exact FD001 contracts if present
# ------------------------------------------------------------------

fd001_exact_objects_present_628hc6a8a1 = bool(
    fd001_presence_628hc6a8a1["exists"].all()
)

if fd001_exact_objects_present_628hc6a8a1:

    fit_raw_628hc6a8a1 = FD001_FIT_RAW_628HC6A3R1
    dev_raw_628hc6a8a1 = FD001_DEV_RAW_628HC6A3R1

    retained_sensors_628hc6a8a1 = list(
        FD001_RETAINED_SENSORS_628HC6A4
    )

    model_features_628hc6a8a1 = list(
        FD001_MODEL_FEATURES_628HC6A4
    )

    print("\n--- FD001 EXACT CONTRACT ---")
    print("FIT raw shape       :", tuple(fit_raw_628hc6a8a1.shape))
    print("DEV raw shape       :", tuple(dev_raw_628hc6a8a1.shape))
    print("Retained sensors    :", retained_sensors_628hc6a8a1)
    print("Model features      :", model_features_628hc6a8a1)

else:

    fit_raw_628hc6a8a1 = None
    dev_raw_628hc6a8a1 = None
    retained_sensors_628hc6a8a1 = None
    model_features_628hc6a8a1 = None


# ------------------------------------------------------------------
# 4. Historical manifest severity semantics
# ------------------------------------------------------------------

manifest_exists_628hc6a8a1 = (
    "FD002_CORRUPTION_MANIFEST_628GD"
    in globals()
)

if manifest_exists_628hc6a8a1:

    manifest_exact_628hc6a8a1 = (
        FD002_CORRUPTION_MANIFEST_628GD.copy()
    )

    manifest_summary_628hc6a8a1 = (
        manifest_exact_628hc6a8a1
        .groupby(
            ["family", "severity"],
            sort=True,
            dropna=False,
        )
        .agg(
            scenarios=("scenario_id", "size"),
            sensors=("sensor", "nunique"),
            alpha_min=("alpha", "min"),
            alpha_median=("alpha", "median"),
            alpha_max=("alpha", "max"),
            requested_fraction_min=("requested_fraction", "min"),
            requested_fraction_median=("requested_fraction", "median"),
            requested_fraction_max=("requested_fraction", "max"),
        )
        .reset_index()
    )

    print("\n--- HISTORICAL FD002 FAMILY × SEVERITY SEMANTICS ---")
    display(manifest_summary_628hc6a8a1)

else:

    manifest_exact_628hc6a8a1 = None

    print(
        "\nFD002_CORRUPTION_MANIFEST_628GD is not present."
    )


# ------------------------------------------------------------------
# 5. Display the exact historical reference objects without changing them
# ------------------------------------------------------------------

reference_object_names_628hc6a8a1 = [
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_FIT_CORRUPTION_REFERENCE_628GB",
]

for ref_name_628hc6a8a1 in reference_object_names_628hc6a8a1:

    print("\n" + "-" * 118)
    print(ref_name_628hc6a8a1)
    print("-" * 118)

    if ref_name_628hc6a8a1 not in globals():

        print("MISSING")
        continue

    ref_obj_628hc6a8a1 = globals()[
        ref_name_628hc6a8a1
    ]

    print("type :", type(ref_obj_628hc6a8a1).__name__)

    if isinstance(ref_obj_628hc6a8a1, pd.DataFrame):

        print("shape:", tuple(ref_obj_628hc6a8a1.shape))
        print("columns:", list(ref_obj_628hc6a8a1.columns))
        display(ref_obj_628hc6a8a1)

    elif isinstance(ref_obj_628hc6a8a1, pd.Series):

        print("shape:", tuple(ref_obj_628hc6a8a1.shape))
        display(ref_obj_628hc6a8a1)

    elif isinstance(ref_obj_628hc6a8a1, dict):

        print("keys:", sorted(list(ref_obj_628hc6a8a1.keys())))

        dict_rows_628hc6a8a1 = []

        for key_628hc6a8a1, value_628hc6a8a1 in ref_obj_628hc6a8a1.items():

            value_array_628hc6a8a1 = np.asarray(
                value_628hc6a8a1
            )

            dict_rows_628hc6a8a1.append(
                {
                    "key": key_628hc6a8a1,
                    "type": type(value_628hc6a8a1).__name__,
                    "shape": tuple(value_array_628hc6a8a1.shape),
                }
            )

        display(
            pd.DataFrame(
                dict_rows_628hc6a8a1
            )
        )

    else:

        print(repr(ref_obj_628hc6a8a1))


# ------------------------------------------------------------------
# 6. Historical operator contract: identity + signatures + dependencies
# ------------------------------------------------------------------

operator_contract_exists_628hc6a8a1 = (
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC"
    in globals()
    and
    isinstance(
        FD002_CORRUPTION_OPERATOR_CONTRACT_628GC,
        dict,
    )
)

operator_rows_628hc6a8a1 = []

if operator_contract_exists_628hc6a8a1:

    for family_628hc6a8a1 in sorted(
        FD002_CORRUPTION_OPERATOR_CONTRACT_628GC.keys()
    ):

        fn_628hc6a8a1 = (
            FD002_CORRUPTION_OPERATOR_CONTRACT_628GC[
                family_628hc6a8a1
            ]
        )

        try:
            signature_628hc6a8a1 = str(
                inspect.signature(
                    fn_628hc6a8a1
                )
            )
        except Exception:
            signature_628hc6a8a1 = (
                "<signature unavailable>"
            )

        dependencies_628hc6a8a1 = (
            list(fn_628hc6a8a1.__code__.co_names)
            if hasattr(fn_628hc6a8a1, "__code__")
            else None
        )

        operator_rows_628hc6a8a1.append(
            {
                "family": family_628hc6a8a1,
                "function_name": getattr(
                    fn_628hc6a8a1,
                    "__name__",
                    type(fn_628hc6a8a1).__name__,
                ),
                "signature": signature_628hc6a8a1,
                "global_dependencies": dependencies_628hc6a8a1,
            }
        )

operator_table_628hc6a8a1 = pd.DataFrame(
    operator_rows_628hc6a8a1
)

print("\n--- HISTORICAL CORRUPTION OPERATOR CONTRACT ---")
display(operator_table_628hc6a8a1)


# ------------------------------------------------------------------
# 7. Print exact source for each operator — inspection only
# ------------------------------------------------------------------

if operator_contract_exists_628hc6a8a1:

    for family_628hc6a8a1 in sorted(
        FD002_CORRUPTION_OPERATOR_CONTRACT_628GC.keys()
    ):

        fn_628hc6a8a1 = (
            FD002_CORRUPTION_OPERATOR_CONTRACT_628GC[
                family_628hc6a8a1
            ]
        )

        print("\n" + "=" * 118)
        print(
            "OPERATOR:",
            family_628hc6a8a1,
            "|",
            getattr(
                fn_628hc6a8a1,
                "__name__",
                type(fn_628hc6a8a1).__name__,
            ),
        )
        print("=" * 118)

        try:

            print(
                inspect.getsource(
                    fn_628hc6a8a1
                )
            )

        except Exception as exc_628hc6a8a1:

            print(
                "Source unavailable:",
                repr(exc_628hc6a8a1),
            )


# ------------------------------------------------------------------
# 8. Safety / readiness audit
# ------------------------------------------------------------------

historical_core_present_628hc6a8a1 = bool(
    historical_presence_628hc6a8a1[
        historical_presence_628hc6a8a1["name"].isin(
            [
                "FD002_CORRUPTION_MANIFEST_628GD",
                "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
                "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
            ]
        )
    ]["exists"].all()
)

audit_628hc6a8a1 = pd.DataFrame(
    [
        (
            "c6_a8_a0_passed",
            bool(
                all_checks_passed_628hc6a8a0
            ),
        ),
        (
            "exact_fd001_fit_raw_present",
            "FD001_FIT_RAW_628HC6A3R1"
            in globals(),
        ),
        (
            "exact_fd001_dev_raw_present",
            "FD001_DEV_RAW_628HC6A3R1"
            in globals(),
        ),
        (
            "exact_fd001_sensor_contract_present",
            "FD001_RETAINED_SENSORS_628HC6A4"
            in globals(),
        ),
        (
            "exact_fd001_feature_contract_present",
            "FD001_MODEL_FEATURES_628HC6A4"
            in globals(),
        ),
        (
            "historical_manifest_present",
            manifest_exists_628hc6a8a1,
        ),
        (
            "historical_operator_contract_present",
            operator_contract_exists_628hc6a8a1,
        ),
        (
            "historical_scale_reference_present",
            "FD002_CORRUPTION_SCALE_REFERENCE_628GB"
            in globals(),
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_corruption_executed",
            True,
        ),
        (
            "no_model_execution",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print("\n--- C6-A8-A1 AUDIT ---")
display(audit_628hc6a8a1)

all_checks_passed_628hc6a8a1 = bool(
    audit_628hc6a8a1[
        "passed"
    ].all()
)

print("\n" + "=" * 118)
print("STEP 6.28H-C6-A8-A1 FINAL SUMMARY")
print("=" * 118)

print(
    "Exact FD001 FIT/DEV objects locked :",
    fd001_exact_objects_present_628hc6a8a1,
)

print(
    "Historical corruption core present :",
    historical_core_present_628hc6a8a1,
)

print(
    "Operator contract recovered         :",
    operator_contract_exists_628hc6a8a1,
)

print("FD001 TEST features read             : False")
print("Official FD001 TEST labels read      : False")
print("Corruption executed                  : False")
print("Model executed/retrained             : False")
print("Threshold modified                   : False")
print("Files written                        : False")

print(
    "All C6-A8-A1 checks passed           :",
    all_checks_passed_628hc6a8a1,
)

print(
    "\nNext: derive the FD001 FIT-only corruption severity/reference "
    "contract from the recovered historical rule. "
    "Do not execute the 315 DEV corruptions yet."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A1 ==="
)

In [ ]:
# ==========================================================================================
# STEP 6.28H-C6-A8-A2 — FD001 FIT-ONLY CORRUPTION REFERENCE CONTRACT
# ==========================================================================================

import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C6-A8-A2 — FD001 FIT-ONLY CORRUPTION REFERENCE CONTRACT")
print("=" * 122)

# ------------------------------------------------------------------------------------------
# 0. Gate
# ------------------------------------------------------------------------------------------

if "all_checks_passed_628hc6a8a1" not in globals():
    raise RuntimeError("C6-A8-A1 completion flag is missing.")

if not bool(all_checks_passed_628hc6a8a1):
    raise RuntimeError("C6-A8-A1 did not pass. Stop before C6-A8-A2.")

required_628hc6a8a2 = [
    "FD001_FIT_RAW_628HC6A3R1",
    "FD001_DEV_RAW_628HC6A3R1",
    "FD001_RETAINED_SENSORS_628HC6A4",
    "FD002_FIT_CORRUPTION_REFERENCE_628GB",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
]

missing_628hc6a8a2 = [
    name for name in required_628hc6a8a2
    if name not in globals()
]

if missing_628hc6a8a2:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_628hc6a8a2)
    )

# ------------------------------------------------------------------------------------------
# 1. Verify exactly what the historical FD002 operators consumed
# ------------------------------------------------------------------------------------------

historical_contract_628hc6a8a2 = (
    FD002_CORRUPTION_OPERATOR_CONTRACT_628GC
)

if "sensor_scale" not in historical_contract_628hc6a8a2:
    raise RuntimeError(
        "Historical operator contract has no sensor_scale mapping."
    )

if "dropout_mean_reference" not in historical_contract_628hc6a8a2:
    raise RuntimeError(
        "Historical operator contract has no dropout_mean_reference mapping."
    )

historical_sensor_scale_628hc6a8a2 = dict(
    historical_contract_628hc6a8a2["sensor_scale"]
)

historical_dropout_mean_628hc6a8a2 = dict(
    historical_contract_628hc6a8a2["dropout_mean_reference"]
)

hist_scale_table_628hc6a8a2 = (
    FD002_CORRUPTION_SCALE_REFERENCE_628GB.copy()
)

hist_dropout_table_628hc6a8a2 = (
    FD002_DROPOUT_IMPUTATION_REFERENCE_628GB.copy()
)

hist_fit_ref_628hc6a8a2 = (
    FD002_FIT_CORRUPTION_REFERENCE_628GB.copy()
)

# ------------------------------------------------------------------------------------------
# 2. Verify historical sensor-scale mapping exactly
# ------------------------------------------------------------------------------------------

scale_mapping_errors_628hc6a8a2 = []

for row_628hc6a8a2 in hist_scale_table_628hc6a8a2.itertuples(
    index=False
):
    sensor_628hc6a8a2 = str(
        row_628hc6a8a2.sensor
    )

    table_value_628hc6a8a2 = float(
        row_628hc6a8a2.pooled_within_regime_std
    )

    contract_value_628hc6a8a2 = float(
        historical_sensor_scale_628hc6a8a2[
            sensor_628hc6a8a2
        ]
    )

    scale_mapping_errors_628hc6a8a2.append(
        abs(
            table_value_628hc6a8a2
            -
            contract_value_628hc6a8a2
        )
    )

max_scale_mapping_error_628hc6a8a2 = float(
    np.max(
        scale_mapping_errors_628hc6a8a2
    )
)

# ------------------------------------------------------------------------------------------
# 3. Verify historical dropout-imputation mapping exactly
# ------------------------------------------------------------------------------------------

dropout_mapping_errors_628hc6a8a2 = []

for row_628hc6a8a2 in hist_dropout_table_628hc6a8a2.itertuples(
    index=False
):
    key_628hc6a8a2 = (
        int(
            row_628hc6a8a2.regime_id
        ),
        str(
            row_628hc6a8a2.sensor
        ),
    )

    table_value_628hc6a8a2 = float(
        row_628hc6a8a2.mean
    )

    contract_value_628hc6a8a2 = float(
        historical_dropout_mean_628hc6a8a2[
            key_628hc6a8a2
        ]
    )

    dropout_mapping_errors_628hc6a8a2.append(
        abs(
            table_value_628hc6a8a2
            -
            contract_value_628hc6a8a2
        )
    )

max_dropout_mapping_error_628hc6a8a2 = float(
    np.max(
        dropout_mapping_errors_628hc6a8a2
    )
)

# ------------------------------------------------------------------------------------------
# 4. Recover historical STD convention rather than assuming ddof
# ------------------------------------------------------------------------------------------

retained_sensors_628hc6a8a2 = list(
    FD001_RETAINED_SENSORS_628HC6A4
)

ddof0_errors_628hc6a8a2 = []
ddof1_errors_628hc6a8a2 = []

for sensor_628hc6a8a2 in retained_sensors_628hc6a8a2:

    frozen_row_628hc6a8a2 = (
        hist_scale_table_628hc6a8a2.loc[
            hist_scale_table_628hc6a8a2["sensor"]
            ==
            sensor_628hc6a8a2
        ]
    )

    if len(frozen_row_628hc6a8a2) != 1:
        raise RuntimeError(
            f"Historical scale row is not unique for "
            f"{sensor_628hc6a8a2}."
        )

    frozen_global_std_628hc6a8a2 = float(
        frozen_row_628hc6a8a2.iloc[
            0
        ][
            "global_raw_std"
        ]
    )

    x_hist_628hc6a8a2 = (
        hist_fit_ref_628hc6a8a2[
            sensor_628hc6a8a2
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    ddof0_errors_628hc6a8a2.append(
        abs(
            float(
                np.std(
                    x_hist_628hc6a8a2,
                    ddof=0,
                )
            )
            -
            frozen_global_std_628hc6a8a2
        )
    )

    ddof1_errors_628hc6a8a2.append(
        abs(
            float(
                np.std(
                    x_hist_628hc6a8a2,
                    ddof=1,
                )
            )
            -
            frozen_global_std_628hc6a8a2
        )
    )

max_ddof0_error_628hc6a8a2 = float(
    np.max(
        ddof0_errors_628hc6a8a2
    )
)

max_ddof1_error_628hc6a8a2 = float(
    np.max(
        ddof1_errors_628hc6a8a2
    )
)

if max_ddof0_error_628hc6a8a2 <= 1e-10:

    HISTORICAL_STD_DDOF_628HC6A8A2 = 0

elif max_ddof1_error_628hc6a8a2 <= 1e-10:

    HISTORICAL_STD_DDOF_628HC6A8A2 = 1

else:

    raise RuntimeError(
        "Could not reproduce historical global_raw_std convention. "
        f"ddof=0 max error={max_ddof0_error_628hc6a8a2:.3e}; "
        f"ddof=1 max error={max_ddof1_error_628hc6a8a2:.3e}"
    )

print("\n--- HISTORICAL REFERENCE IDENTITY ---")

print(
    "sensor_scale mapping max error :",
    max_scale_mapping_error_628hc6a8a2,
)

print(
    "dropout mean mapping max error :",
    max_dropout_mapping_error_628hc6a8a2,
)

print(
    "historical STD ddof             :",
    HISTORICAL_STD_DDOF_628HC6A8A2,
)

print(
    "ddof=0 max error                :",
    max_ddof0_error_628hc6a8a2,
)

print(
    "ddof=1 max error                :",
    max_ddof1_error_628hc6a8a2,
)

# ------------------------------------------------------------------------------------------
# 5. Build canonical FD001 FIT/DEV corruption bases
#
# FD001 has one operating condition.
# Therefore its condition-aware corruption reference has one regime.
#
# IMPORTANT:
#   FIT is used to derive all numerical corruption references.
#   DEV is only prepared as a future corruption target.
# ------------------------------------------------------------------------------------------

fit_raw_628hc6a8a2 = (
    FD001_FIT_RAW_628HC6A3R1.copy()
)

dev_raw_628hc6a8a2 = (
    FD001_DEV_RAW_628HC6A3R1.copy()
)

if "unit" not in fit_raw_628hc6a8a2.columns:
    raise RuntimeError(
        "FD001 FIT raw object is missing 'unit'."
    )

if "unit" not in dev_raw_628hc6a8a2.columns:
    raise RuntimeError(
        "FD001 DEV raw object is missing 'unit'."
    )

if "cycle" not in fit_raw_628hc6a8a2.columns:
    raise RuntimeError(
        "FD001 FIT raw object is missing 'cycle'."
    )

if "cycle" not in dev_raw_628hc6a8a2.columns:
    raise RuntimeError(
        "FD001 DEV raw object is missing 'cycle'."
    )

missing_fit_sensors_628hc6a8a2 = [
    sensor
    for sensor in retained_sensors_628hc6a8a2
    if sensor not in fit_raw_628hc6a8a2.columns
]

missing_dev_sensors_628hc6a8a2 = [
    sensor
    for sensor in retained_sensors_628hc6a8a2
    if sensor not in dev_raw_628hc6a8a2.columns
]

if missing_fit_sensors_628hc6a8a2:
    raise RuntimeError(
        "FD001 FIT missing retained sensors: "
        + ", ".join(
            missing_fit_sensors_628hc6a8a2
        )
    )

if missing_dev_sensors_628hc6a8a2:
    raise RuntimeError(
        "FD001 DEV missing retained sensors: "
        + ", ".join(
            missing_dev_sensors_628hc6a8a2
        )
    )

FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2 = (
    fit_raw_628hc6a8a2
    .rename(
        columns={
            "unit": "unit_id"
        }
    )
    .copy()
)

FD001_DEV_CORRUPTION_BASE_628HC6A8A2 = (
    dev_raw_628hc6a8a2
    .rename(
        columns={
            "unit": "unit_id"
        }
    )
    .copy()
)

FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2[
    "regime_id"
] = 0

FD001_DEV_CORRUPTION_BASE_628HC6A8A2[
    "regime_id"
] = 0

# ------------------------------------------------------------------------------------------
# 6. Derive FD001 FIT-only sensor scale
# ------------------------------------------------------------------------------------------

scale_rows_628hc6a8a2 = []

for sensor_628hc6a8a2 in retained_sensors_628hc6a8a2:

    values_628hc6a8a2 = (
        FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2[
            sensor_628hc6a8a2
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    global_std_628hc6a8a2 = float(
        np.std(
            values_628hc6a8a2,
            ddof=HISTORICAL_STD_DDOF_628HC6A8A2,
        )
    )

    # One FD001 operating regime:
    # pooled within-regime STD == global STD.
    pooled_within_std_628hc6a8a2 = (
        global_std_628hc6a8a2
    )

    if (
        not np.isfinite(
            pooled_within_std_628hc6a8a2
        )
        or
        pooled_within_std_628hc6a8a2 <= 0.0
    ):
        raise RuntimeError(
            f"Invalid FIT corruption scale for "
            f"{sensor_628hc6a8a2}: "
            f"{pooled_within_std_628hc6a8a2}"
        )

    scale_rows_628hc6a8a2.append(
        {
            "sensor":
                sensor_628hc6a8a2,

            "pooled_within_regime_std":
                pooled_within_std_628hc6a8a2,

            "global_raw_std":
                global_std_628hc6a8a2,

            "global_to_within_ratio":
                (
                    global_std_628hc6a8a2
                    /
                    pooled_within_std_628hc6a8a2
                ),
        }
    )

FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2 = (
    pd.DataFrame(
        scale_rows_628hc6a8a2
    )
)

SENSOR_SCALE_FD001_628HC6A8A2 = {
    str(
        row.sensor
    ):
    float(
        row.pooled_within_regime_std
    )

    for row
    in
    FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2.itertuples(
        index=False
    )
}

# ------------------------------------------------------------------------------------------
# 7. Derive FD001 FIT-only dropout means
# ------------------------------------------------------------------------------------------

dropout_rows_628hc6a8a2 = []

for sensor_628hc6a8a2 in retained_sensors_628hc6a8a2:

    mean_628hc6a8a2 = float(
        FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2[
            sensor_628hc6a8a2
        ].mean()
    )

    if not np.isfinite(
        mean_628hc6a8a2
    ):
        raise RuntimeError(
            f"Invalid FIT dropout mean for "
            f"{sensor_628hc6a8a2}."
        )

    dropout_rows_628hc6a8a2.append(
        {
            "regime_id": 0,
            "sensor": sensor_628hc6a8a2,
            "mean": mean_628hc6a8a2,
        }
    )

FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2 = (
    pd.DataFrame(
        dropout_rows_628hc6a8a2
    )
)

DROPOUT_MEAN_FD001_628HC6A8A2 = {
    (
        int(
            row.regime_id
        ),
        str(
            row.sensor
        ),
    ):
    float(
        row.mean
    )

    for row
    in
    FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2.itertuples(
        index=False
    )
}

# ------------------------------------------------------------------------------------------
# 8. Display derived references
# ------------------------------------------------------------------------------------------

print(
    "\n--- FD001 FIT-ONLY CORRUPTION SCALE REFERENCE ---"
)

display(
    FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2
)

print(
    "\n--- FD001 FIT-ONLY DROPOUT IMPUTATION REFERENCE ---"
)

display(
    FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2
)

print(
    "\n--- FD001 CORRUPTION BASE CONTRACT ---"
)

print(
    "FIT reference shape :",
    tuple(
        FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2.shape
    ),
)

print(
    "DEV base shape      :",
    tuple(
        FD001_DEV_CORRUPTION_BASE_628HC6A8A2.shape
    ),
)

print(
    "FIT regime IDs      :",
    sorted(
        FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2[
            "regime_id"
        ]
        .unique()
        .tolist()
    ),
)

print(
    "DEV regime IDs      :",
    sorted(
        FD001_DEV_CORRUPTION_BASE_628HC6A8A2[
            "regime_id"
        ]
        .unique()
        .tolist()
    ),
)

# ------------------------------------------------------------------------------------------
# 9. Audit
# ------------------------------------------------------------------------------------------

audit_628hc6a8a2 = pd.DataFrame(
    [
        (
            "previous_step_passed",
            bool(
                all_checks_passed_628hc6a8a1
            ),
        ),

        (
            "historical_sensor_scale_mapping_exact",
            bool(
                max_scale_mapping_error_628hc6a8a2
                <=
                1e-14
            ),
        ),

        (
            "historical_dropout_mapping_exact",
            bool(
                max_dropout_mapping_error_628hc6a8a2
                <=
                1e-14
            ),
        ),

        (
            "historical_std_convention_recovered",
            HISTORICAL_STD_DDOF_628HC6A8A2
            in
            (
                0,
                1,
            ),
        ),

        (
            "fd001_fit_reference_rows_16779",
            len(
                FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2
            )
            ==
            16779,
        ),

        (
            "fd001_dev_base_rows_3852",
            len(
                FD001_DEV_CORRUPTION_BASE_628HC6A8A2
            )
            ==
            3852,
        ),

        (
            "fd001_single_regime_fit",
            FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2[
                "regime_id"
            ].nunique()
            ==
            1,
        ),

        (
            "fd001_single_regime_dev",
            FD001_DEV_CORRUPTION_BASE_628HC6A8A2[
                "regime_id"
            ].nunique()
            ==
            1,
        ),

        (
            "fd001_scale_rows_15",
            len(
                FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2
            )
            ==
            15,
        ),

        (
            "fd001_dropout_reference_rows_15",
            len(
                FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2
            )
            ==
            15,
        ),

        (
            "fd001_sensor_scales_positive_finite",
            bool(
                np.isfinite(
                    FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2[
                        "pooled_within_regime_std"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                ).all()
                and
                (
                    FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2[
                        "pooled_within_regime_std"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                    >
                    0.0
                ).all()
            ),
        ),

        (
            "fd001_dropout_means_finite",
            bool(
                np.isfinite(
                    FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2[
                        "mean"
                    ]
                    .to_numpy(
                        dtype=np.float64
                    )
                ).all()
            ),
        ),

        (
            "test_features_not_read",
            True,
        ),

        (
            "official_test_labels_not_read",
            True,
        ),

        (
            "test_not_scored",
            True,
        ),

        (
            "no_dev_statistics_used_for_reference_fitting",
            True,
        ),

        (
            "no_corruption_executed",
            True,
        ),

        (
            "no_model_execution",
            True,
        ),

        (
            "no_model_retraining",
            True,
        ),

        (
            "no_threshold_modification",
            True,
        ),

        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

print(
    "\n--- C6-A8-A2 AUDIT ---"
)

display(
    audit_628hc6a8a2
)

all_checks_passed_628hc6a8a2 = bool(
    audit_628hc6a8a2[
        "passed"
    ].all()
)

# ------------------------------------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 122)
print("STEP 6.28H-C6-A8-A2 FINAL SUMMARY")
print("=" * 122)

print(
    "Historical sensor-scale identity verified :",
    max_scale_mapping_error_628hc6a8a2
    <=
    1e-14,
)

print(
    "Historical dropout-mean identity verified :",
    max_dropout_mapping_error_628hc6a8a2
    <=
    1e-14,
)

print(
    "Historical STD ddof                      :",
    HISTORICAL_STD_DDOF_628HC6A8A2,
)

print(
    "FD001 corruption regimes                 : 1"
)

print(
    "FD001 sensor-scale reference rows        :",
    len(
        FD001_CORRUPTION_SCALE_REFERENCE_628HC6A8A2
    ),
)

print(
    "FD001 dropout-imputation rows            :",
    len(
        FD001_DROPOUT_IMPUTATION_REFERENCE_628HC6A8A2
    ),
)

print(
    "Reference fitting source                 : FD001 FIT only"
)

print(
    "DEV statistics used for reference fitting: False"
)

print(
    "FD001 TEST features read                 : False"
)

print(
    "Official FD001 TEST labels read          : False"
)

print(
    "Corruption executed                      : False"
)

print(
    "Model executed/retrained                 : False"
)

print(
    "Threshold modified                       : False"
)

print(
    "Files written                            : False"
)

print(
    "All C6-A8-A2 checks passed               :",
    all_checks_passed_628hc6a8a2,
)

if not all_checks_passed_628hc6a8a2:
    raise RuntimeError(
        "C6-A8-A2 did not pass. "
        "Do not build the FD001 corruption manifest yet."
    )

print(
    "\nNext required step: C6-A8-A3 — build the FD001 "
    "315-scenario manifest and FD001-specific corruption "
    "operator bridge using the frozen family/severity semantics "
    "and these FIT-only references."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A2 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A8-A3 — FD001 315-SCENARIO MANIFEST + CORRUPTION OPERATOR BRIDGE LOCK
# ======================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A8-A3 — FD001 315-SCENARIO MANIFEST + CORRUPTION OPERATOR BRIDGE LOCK")
print("=" * 118)

# ----------------------------- 0. Gate -----------------------------------------------

required = [
    "all_checks_passed_628hc6a8a2",
    "FD001_DEV_CORRUPTION_BASE_628HC6A8A2",
    "FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2",
    "FD001_RETAINED_SENSORS_628HC6A4",
    "SENSOR_SCALE_FD001_628HC6A8A2",
    "DROPOUT_MEAN_FD001_628HC6A8A2",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
    )

if not bool(all_checks_passed_628hc6a8a2):
    raise RuntimeError(
        "C6-A8-A2 did not pass."
    )

# ---------------------- 1. Canonical FD001 bases -------------------------------------

FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A3 = (
    FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A2
    .copy(deep=True)
    .reset_index(drop=True)
)

FD001_DEV_CORRUPTION_BASE_628HC6A8A3 = (
    FD001_DEV_CORRUPTION_BASE_628HC6A8A2
    .copy(deep=True)
    .reset_index(drop=True)
)

FD001_RETAINED_SENSORS_628HC6A8A3 = list(
    FD001_RETAINED_SENSORS_628HC6A4
)

SENSOR_SCALE_FD001_628HC6A8A3 = {
    str(k): float(v)
    for k, v in SENSOR_SCALE_FD001_628HC6A8A2.items()
}

DROPOUT_MEAN_FD001_628HC6A8A3 = {
    (int(k[0]), str(k[1])): float(v)
    for k, v in DROPOUT_MEAN_FD001_628HC6A8A2.items()
}

# --------------------- 2. Transfer frozen 315-scenario manifest ----------------------

src = (
    FD002_CORRUPTION_MANIFEST_628GD
    .copy(deep=True)
    .reset_index(drop=True)
)

manifest_cols = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

missing_cols = [
    c for c in manifest_cols
    if c not in src.columns
]

if missing_cols:
    raise RuntimeError(
        "Historical manifest missing columns: "
        + ", ".join(missing_cols)
    )

if len(src) != 315:
    raise RuntimeError(
        f"Historical manifest must contain 315 rows; got {len(src)}."
    )

FD001_CORRUPTION_MANIFEST_628HC6A8A3 = (
    src[manifest_cols]
    .copy(deep=True)
)

# Preserve exact scenario order and parameters.
# Only textual scenario ID becomes FD001-specific.
FD001_CORRUPTION_MANIFEST_628HC6A8A3[
    "scenario_id"
] = [
    f"FD001_SCENARIO_{int(n):03d}"
    for n in
    FD001_CORRUPTION_MANIFEST_628HC6A8A3[
        "scenario_number"
    ].to_numpy()
]

# --------------------- 3. Frozen-semantics verification -------------------------------

expected_family_counts = {
    "bias": 90,
    "drift": 90,
    "noise": 45,
    "dropout": 45,
    "stuck": 45,
}

actual_family_counts = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3[
        "family"
    ]
    .value_counts()
    .to_dict()
)

severity_contract = {
    ("bias", "low"): ("alpha", 0.25),
    ("bias", "medium"): ("alpha", 0.50),
    ("bias", "high"): ("alpha", 1.00),

    ("drift", "low"): ("alpha", 0.25),
    ("drift", "medium"): ("alpha", 0.50),
    ("drift", "high"): ("alpha", 1.00),

    ("noise", "low"): ("alpha", 0.10),
    ("noise", "medium"): ("alpha", 0.25),
    ("noise", "high"): ("alpha", 0.50),

    ("dropout", "low"): ("requested_fraction", 0.10),
    ("dropout", "medium"): ("requested_fraction", 0.25),
    ("dropout", "high"): ("requested_fraction", 0.50),

    ("stuck", "low"): ("requested_fraction", 0.20),
    ("stuck", "medium"): ("requested_fraction", 0.40),
    ("stuck", "high"): ("requested_fraction", 0.60),
}

severity_semantics_ok = True

for (family, severity), (col, expected) in severity_contract.items():

    vals = (
        FD001_CORRUPTION_MANIFEST_628HC6A8A3.loc[
            (
                FD001_CORRUPTION_MANIFEST_628HC6A8A3["family"]
                ==
                family
            )
            &
            (
                FD001_CORRUPTION_MANIFEST_628HC6A8A3["severity"]
                ==
                severity
            ),
            col,
        ]
        .to_numpy(dtype=np.float64)
    )

    if (
        len(vals) == 0
        or
        not np.isfinite(vals).all()
        or
        not np.all(vals == expected)
    ):
        severity_semantics_ok = False
        break

retained_set = set(
    FD001_RETAINED_SENSORS_628HC6A8A3
)

manifest_sensor_set = set(
    FD001_CORRUPTION_MANIFEST_628HC6A8A3[
        "sensor"
    ]
    .astype(str)
    .tolist()
)

bias_drift = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3[
        FD001_CORRUPTION_MANIFEST_628HC6A8A3[
            "family"
        ].isin(
            ["bias", "drift"]
        )
    ]
)

direction_pairs_ok = True

for _, g in bias_drift.groupby(
    [
        "family",
        "severity",
        "sensor",
    ],
    sort=False,
):

    if (
        len(g) != 2
        or
        set(
            g[
                "direction"
            ]
            .astype(int)
            .tolist()
        )
        !=
        {-1, 1}
    ):
        direction_pairs_ok = False
        break

# ----------------------- 4. FD001-specific operators ----------------------------------

def validate_fd001_corruption_base_628hc6a8a3(
    df,
    sensor,
):

    if not isinstance(
        df,
        pd.DataFrame,
    ):
        raise TypeError(
            "Corruption base must be a pandas DataFrame."
        )

    if (
        sensor
        not in
        FD001_RETAINED_SENSORS_628HC6A8A3
    ):
        raise ValueError(
            f"Unlocked FD001 sensor: {sensor}"
        )

    required_cols = {
        "unit_id",
        "cycle",
        "regime_id",
        sensor,
    }

    missing_local = sorted(
        required_cols.difference(
            df.columns
        )
    )

    if missing_local:
        raise ValueError(
            "Missing columns: "
            + ", ".join(missing_local)
        )

    if not np.array_equal(
        df.index.to_numpy(),
        np.arange(
            len(df)
        ),
    ):
        raise ValueError(
            "Corruption base must use RangeIndex 0..N-1."
        )


def apply_fd001_bias_628hc6a8a3(
    df,
    sensor,
    alpha,
    direction,
):

    validate_fd001_corruption_base_628hc6a8a3(
        df,
        sensor,
    )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    amount = float(
        direction
        *
        alpha
        *
        SENSOR_SCALE_FD001_628HC6A8A3[
            sensor
        ]
    )

    out = df.copy()

    out[
        sensor
    ] = (
        out[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        amount
    )

    return (
        out,
        amount,
    )


def apply_fd001_noise_628hc6a8a3(
    df,
    sensor,
    alpha,
    seed=42,
):

    validate_fd001_corruption_base_628hc6a8a3(
        df,
        sensor,
    )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    target_std = float(
        alpha
        *
        SENSOR_SCALE_FD001_628HC6A8A3[
            sensor
        ]
    )

    rng = np.random.default_rng(
        int(seed)
    )

    noise = (
        rng.normal(
            0.0,
            1.0,
            size=len(df),
        )
        *
        target_std
    )

    out = df.copy()

    out[
        sensor
    ] = (
        out[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        noise
    )

    return (
        out,
        noise,
        target_std,
    )


def apply_fd001_drift_628hc6a8a3(
    df,
    sensor,
    alpha,
    direction,
):

    validate_fd001_corruption_base_628hc6a8a3(
        df,
        sensor,
    )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    final_drift = float(
        direction
        *
        alpha
        *
        SENSOR_SCALE_FD001_628HC6A8A3[
            sensor
        ]
    )

    out = df.copy()

    drift = np.zeros(
        len(out),
        dtype=np.float64,
    )

    for _, group in out.groupby(
        "unit_id",
        sort=False,
    ):

        group = group.sort_values(
            "cycle"
        )

        pos = (
            group.index
            .to_numpy(
                dtype=np.int64
            )
        )

        if len(pos) == 1:

            drift[
                pos
            ] = final_drift

        else:

            drift[
                pos
            ] = np.linspace(
                0.0,
                final_drift,
                len(pos),
                endpoint=True,
                dtype=np.float64,
            )

    out[
        sensor
    ] = (
        out[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        drift
    )

    return (
        out,
        drift,
        final_drift,
    )


def apply_fd001_dropout_628hc6a8a3(
    df,
    sensor,
    dropout_rate,
    seed=42,
):

    validate_fd001_corruption_base_628hc6a8a3(
        df,
        sensor,
    )

    rate = float(
        dropout_rate
    )

    if not (
        0.0
        <=
        rate
        <
        1.0
    ):
        raise ValueError(
            "dropout_rate must satisfy 0 <= rate < 1."
        )

    out = df.copy()

    n = len(
        out
    )

    n_drop = int(
        round(
            rate
            *
            n
        )
    )

    rng = np.random.default_rng(
        int(seed)
    )

    drop_pos = (
        rng.permutation(
            n
        )[
            :n_drop
        ]
    )

    mask = np.zeros(
        n,
        dtype=bool,
    )

    mask[
        drop_pos
    ] = True

    sensor_pos = (
        out.columns
        .get_loc(
            sensor
        )
    )

    out.iloc[
        drop_pos,
        sensor_pos,
    ] = np.nan

    regimes = (
        out[
            "regime_id"
        ]
        .to_numpy()
    )

    impute = np.empty(
        n,
        dtype=np.float64,
    )

    for i in range(
        n
    ):

        key = (
            int(
                regimes[
                    i
                ]
            ),
            sensor,
        )

        if (
            key
            not in
            DROPOUT_MEAN_FD001_628HC6A8A3
        ):
            raise RuntimeError(
                f"Missing dropout mean for "
                f"regime={key[0]}, sensor={sensor}."
            )

        impute[
            i
        ] = (
            DROPOUT_MEAN_FD001_628HC6A8A3[
                key
            ]
        )

    missing_pos = np.flatnonzero(
        mask
    )

    out.iloc[
        missing_pos,
        sensor_pos,
    ] = impute[
        missing_pos
    ]

    return (
        out,
        mask,
        impute,
    )


def apply_fd001_stuck_628hc6a8a3(
    df,
    sensor,
    affected_fraction,
):

    validate_fd001_corruption_base_628hc6a8a3(
        df,
        sensor,
    )

    fraction = float(
        affected_fraction
    )

    if not (
        0.0
        <
        fraction
        <
        1.0
    ):
        raise ValueError(
            "affected_fraction must satisfy 0 < fraction < 1."
        )

    out = df.copy()

    mask = np.zeros(
        len(out),
        dtype=bool,
    )

    records = []

    for unit_id, group in out.groupby(
        "unit_id",
        sort=False,
    ):

        group = group.sort_values(
            "cycle"
        )

        pos = (
            group.index
            .to_numpy(
                dtype=np.int64
            )
        )

        n = len(
            pos
        )

        if n < 2:
            raise RuntimeError(
                "Stuck corruption requires >=2 cycles per engine."
            )

        n_aff = min(
            max(
                1,
                int(
                    round(
                        fraction
                        *
                        n
                    )
                ),
            ),
            n - 1,
        )

        onset = (
            n
            -
            n_aff
        )

        source_pos = pos[
            onset - 1
        ]

        affected_pos = pos[
            onset:
        ]

        freeze_value = float(
            out.loc[
                source_pos,
                sensor,
            ]
        )

        out.loc[
            affected_pos,
            sensor,
        ] = freeze_value

        mask[
            affected_pos
        ] = True

        records.append(
            {
                "unit_id":
                    unit_id,

                "trajectory_length":
                    n,

                "requested_fraction":
                    fraction,

                "affected_count":
                    n_aff,

                "actual_fraction":
                    n_aff / n,

                "source_cycle":
                    float(
                        group.iloc[
                            onset - 1
                        ][
                            "cycle"
                        ]
                    ),

                "stuck_start_cycle":
                    float(
                        group.iloc[
                            onset
                        ][
                            "cycle"
                        ]
                    ),

                "freeze_value":
                    freeze_value,
            }
        )

    return (
        out,
        mask,
        pd.DataFrame(
            records
        ),
    )


historical_smoke_sensor = str(
    FD002_CORRUPTION_OPERATOR_CONTRACT_628GC[
        "smoke_sensor"
    ]
)

if (
    historical_smoke_sensor
    not in
    retained_set
):
    raise RuntimeError(
        "Historical smoke sensor is absent from "
        "the FD001 retained-sensor set."
    )

FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3 = {
    "all_contract_checks_passed":
        False,

    "bias":
        apply_fd001_bias_628hc6a8a3,

    "drift":
        apply_fd001_drift_628hc6a8a3,

    "dropout":
        apply_fd001_dropout_628hc6a8a3,

    "dropout_mean_reference":
        DROPOUT_MEAN_FD001_628HC6A8A3,

    "noise":
        apply_fd001_noise_628hc6a8a3,

    "noise_seed":
        int(
            FD002_CORRUPTION_OPERATOR_CONTRACT_628GC[
                "noise_seed"
            ]
        ),

    "sensor_scale":
        SENSOR_SCALE_FD001_628HC6A8A3,

    "smoke_sensor":
        historical_smoke_sensor,

    "stuck":
        apply_fd001_stuck_628hc6a8a3,
}

# ---------------------- 5. Exact scenario dispatcher ----------------------------------

def apply_fd001_scenario_628hc6a8a3(
    base_raw,
    scenario_row,
):

    family = str(
        scenario_row[
            "family"
        ]
    )

    sensor = str(
        scenario_row[
            "sensor"
        ]
    )

    c = (
        FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3
    )

    if family == "bias":

        return c[
            "bias"
        ](
            base_raw,
            sensor,
            float(
                scenario_row[
                    "alpha"
                ]
            ),
            int(
                scenario_row[
                    "direction"
                ]
            ),
        )[0]

    if family == "noise":

        return c[
            "noise"
        ](
            base_raw,
            sensor,
            float(
                scenario_row[
                    "alpha"
                ]
            ),
            int(
                scenario_row[
                    "seed"
                ]
            ),
        )[0]

    if family == "drift":

        return c[
            "drift"
        ](
            base_raw,
            sensor,
            float(
                scenario_row[
                    "alpha"
                ]
            ),
            int(
                scenario_row[
                    "direction"
                ]
            ),
        )[0]

    if family == "dropout":

        return c[
            "dropout"
        ](
            base_raw,
            sensor,
            float(
                scenario_row[
                    "requested_fraction"
                ]
            ),
            int(
                scenario_row[
                    "seed"
                ]
            ),
        )[0]

    if family == "stuck":

        return c[
            "stuck"
        ](
            base_raw,
            sensor,
            float(
                scenario_row[
                    "requested_fraction"
                ]
            ),
        )[0]

    raise ValueError(
        "Unknown corruption family: "
        + family
    )

# ----------------------------- 6. Audit -----------------------------------------------

expected_operator_keys = {
    "all_contract_checks_passed",
    "bias",
    "drift",
    "dropout",
    "dropout_mean_reference",
    "noise",
    "noise_seed",
    "sensor_scale",
    "smoke_sensor",
    "stuck",
}

audit_628hc6a8a3 = pd.DataFrame(
    [
        (
            "previous_step_passed",
            bool(
                all_checks_passed_628hc6a8a2
            ),
        ),

        (
            "manifest_shape_315x9",
            FD001_CORRUPTION_MANIFEST_628HC6A8A3.shape
            ==
            (315, 9),
        ),

        (
            "scenario_numbers_unique",
            FD001_CORRUPTION_MANIFEST_628HC6A8A3[
                "scenario_number"
            ].nunique()
            ==
            315,
        ),

        (
            "scenario_ids_unique",
            FD001_CORRUPTION_MANIFEST_628HC6A8A3[
                "scenario_id"
            ].nunique()
            ==
            315,
        ),

        (
            "family_counts_exact",
            actual_family_counts
            ==
            expected_family_counts,
        ),

        (
            "severity_semantics_exact",
            severity_semantics_ok,
        ),

        (
            "manifest_sensor_set_exact",
            manifest_sensor_set
            ==
            retained_set,
        ),

        (
            "bias_drift_direction_pairs_exact",
            direction_pairs_ok,
        ),

        (
            "fit_reference_canonical_index",
            np.array_equal(
                FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A3.index.to_numpy(),
                np.arange(
                    len(
                        FD001_FIT_CORRUPTION_REFERENCE_628HC6A8A3
                    )
                ),
            ),
        ),

        (
            "dev_base_canonical_index",
            np.array_equal(
                FD001_DEV_CORRUPTION_BASE_628HC6A8A3.index.to_numpy(),
                np.arange(
                    len(
                        FD001_DEV_CORRUPTION_BASE_628HC6A8A3
                    )
                ),
            ),
        ),

        (
            "sensor_scale_keys_exact",
            set(
                SENSOR_SCALE_FD001_628HC6A8A3
            )
            ==
            retained_set,
        ),

        (
            "dropout_sensor_keys_exact",
            {
                sensor
                for _, sensor
                in DROPOUT_MEAN_FD001_628HC6A8A3
            }
            ==
            retained_set,
        ),

        (
            "dropout_single_regime",
            {
                regime
                for regime, _
                in DROPOUT_MEAN_FD001_628HC6A8A3
            }
            ==
            {0},
        ),

        (
            "operator_key_set_exact",
            set(
                FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3
            )
            ==
            expected_operator_keys,
        ),

        ("test_features_not_read", True),
        ("official_test_labels_not_read", True),
        ("test_not_scored", True),
        ("no_corruption_executed", True),
        ("no_model_execution", True),
        ("no_model_retraining", True),
        ("no_threshold_modification", True),
        ("no_files_written", True),
    ],
    columns=[
        "check",
        "passed",
    ],
)

all_checks_passed_628hc6a8a3 = bool(
    audit_628hc6a8a3[
        "passed"
    ].all()
)

FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3[
    "all_contract_checks_passed"
] = all_checks_passed_628hc6a8a3

print(
    "\n--- FAMILY COUNTS ---"
)

display(
    FD001_CORRUPTION_MANIFEST_628HC6A8A3[
        "family"
    ]
    .value_counts()
    .rename_axis(
        "family"
    )
    .reset_index(
        name="scenarios"
    )
)

print(
    "\n--- FAMILY × SEVERITY COUNTS ---"
)

display(
    FD001_CORRUPTION_MANIFEST_628HC6A8A3
    .groupby(
        [
            "family",
            "severity",
        ],
        sort=True,
    )
    .size()
    .rename(
        "scenarios"
    )
    .reset_index()
)

print(
    "\n--- C6-A8-A3 AUDIT ---"
)

display(
    audit_628hc6a8a3
)

print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A8-A3 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "FD001 manifest shape                 :",
    FD001_CORRUPTION_MANIFEST_628HC6A8A3.shape,
)

print(
    "Family counts exact                  :",
    actual_family_counts
    ==
    expected_family_counts,
)

print(
    "Severity semantics exact             :",
    severity_semantics_ok,
)

print(
    "Retained sensor set exact            :",
    manifest_sensor_set
    ==
    retained_set,
)

print(
    "Bias/drift direction pairs exact     :",
    direction_pairs_ok,
)

print(
    "FD001 sensor-scale entries           :",
    len(
        SENSOR_SCALE_FD001_628HC6A8A3
    ),
)

print(
    "FD001 dropout-reference entries      :",
    len(
        DROPOUT_MEAN_FD001_628HC6A8A3
    ),
)

print(
    "Canonical DEV corruption base shape  :",
    FD001_DEV_CORRUPTION_BASE_628HC6A8A3.shape,
)

print(
    "Operator bridge locked               :",
    FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3[
        "all_contract_checks_passed"
    ],
)

print(
    "Corruptions executed                 : False"
)

print(
    "FD001 TEST features read             : False"
)

print(
    "Official FD001 TEST labels read      : False"
)

print(
    "Model executed/retrained             : False"
)

print(
    "Threshold modified                   : False"
)

print(
    "Files written                        : False"
)

print(
    "All C6-A8-A3 checks passed           :",
    all_checks_passed_628hc6a8a3,
)

if not all_checks_passed_628hc6a8a3:
    raise RuntimeError(
        "C6-A8-A3 failed. "
        "Do not execute corruption scenarios."
    )

print(
    "\nNext: C6-A8-A4 — five-family smoke audit on COPIES "
    "of the canonical FD001 DEV base, then lock the FD001 "
    "normalization/window bridge before the full 315-scenario run."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A3 ==="
)

In [ ]:
# ======================================================================================
# STEP 6.28H-C6-A8-A3-S1 — HISTORICAL CORRUPTION OPERATOR SEMANTIC SOURCE LOCK
# ======================================================================================

import inspect
import hashlib
import pandas as pd

print("=" * 116)
print("STEP 6.28H-C6-A8-A3-S1 — HISTORICAL CORRUPTION OPERATOR SEMANTIC SOURCE LOCK")
print("=" * 116)

if "all_checks_passed_628hc6a8a3" not in globals():
    raise RuntimeError("C6-A8-A3 state is missing.")

if not bool(all_checks_passed_628hc6a8a3):
    raise RuntimeError("C6-A8-A3 must pass first.")

if "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC" not in globals():
    raise RuntimeError("Historical FD002 operator contract is missing.")

contract_628hc6a8a3s1 = FD002_CORRUPTION_OPERATOR_CONTRACT_628GC

families_628hc6a8a3s1 = [
    "bias",
    "drift",
    "noise",
    "dropout",
    "stuck",
]

rows_628hc6a8a3s1 = []

for family in families_628hc6a8a3s1:
    fn = contract_628hc6a8a3s1.get(family, None)

    if not callable(fn):
        raise RuntimeError(
            f"Historical operator for family={family!r} is not callable."
        )

    try:
        source = inspect.getsource(fn)
        source_ok = True
    except Exception as exc:
        source = (
            f"<SOURCE UNAVAILABLE: "
            f"{type(exc).__name__}: {exc}>"
        )
        source_ok = False

    source_sha256 = hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()

    rows_628hc6a8a3s1.append(
        {
            "family": family,
            "function_name": getattr(fn, "__name__", type(fn).__name__),
            "signature": str(inspect.signature(fn)),
            "source_available": source_ok,
            "source_sha256": source_sha256,
        }
    )

    print("\n" + "-" * 116)
    print(f"FAMILY: {family}")
    print("-" * 116)
    print("function :", getattr(fn, "__name__", type(fn).__name__))
    print("signature:", inspect.signature(fn))
    print(
        "co_names :",
        getattr(
            getattr(fn, "__code__", None),
            "co_names",
            None,
        ),
    )
    print("\nSOURCE:\n")
    print(source)

FD002_OPERATOR_SOURCE_AUDIT_628HC6A8A3S1 = pd.DataFrame(
    rows_628hc6a8a3s1
)

print("\n--- CONTRACT AUXILIARIES ---")

print(
    "sensor_scale type       :",
    type(
        contract_628hc6a8a3s1.get(
            "sensor_scale"
        )
    ).__name__,
)

print(
    "dropout_mean_ref type   :",
    type(
        contract_628hc6a8a3s1.get(
            "dropout_mean_reference"
        )
    ).__name__,
)

print(
    "noise_seed              :",
    contract_628hc6a8a3s1.get(
        "noise_seed"
    ),
)

print(
    "smoke_sensor            :",
    contract_628hc6a8a3s1.get(
        "smoke_sensor"
    ),
)

print("\n--- SOURCE AUDIT ---")
display(
    FD002_OPERATOR_SOURCE_AUDIT_628HC6A8A3S1
)

all_sources_available_628hc6a8a3s1 = bool(
    FD002_OPERATOR_SOURCE_AUDIT_628HC6A8A3S1[
        "source_available"
    ].all()
)

print("\n" + "=" * 116)
print("SEMANTIC SOURCE LOCK SUMMARY")
print("=" * 116)

print(
    "All five historical operator sources available :",
    all_sources_available_628hc6a8a3s1,
)

print("Corruption executed                            : False")
print("Model executed                                 : False")
print("MC dropout executed                            : False")
print("FD001 TEST features read                       : False")
print("Official FD001 TEST labels read                : False")
print("Threshold modified                             : False")
print("Files written                                  : False")

if not all_sources_available_628hc6a8a3s1:
    raise RuntimeError(
        "At least one historical operator source could not be recovered. "
        "Do not execute FD001 corruption smoke tests yet."
    )

print(
    "\nPASS — historical operator formulas recovered. "
    "Next step is the five-family FD001 smoke audit."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A3-S1 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A4 — FD001 FIVE-FAMILY CORRUPTION SMOKE / SEMANTIC AUDIT
# ==================================================================================

import numpy as np
import pandas as pd

print("=" * 110)
print("STEP 6.28H-C6-A8-A4 — FD001 FIVE-FAMILY CORRUPTION SMOKE / SEMANTIC AUDIT")
print("=" * 110)

# ------------------------------------------------------------------
# 0. Prior locks
# ------------------------------------------------------------------
if not bool(globals().get("all_checks_passed_628hc6a8a3", False)):
    raise RuntimeError("C6-A8-A3 must pass first.")

src_audit = globals().get(
    "FD001_OPERATOR_SOURCE_AUDIT_628HC6A8A3S1"
)

if (
    not isinstance(src_audit, pd.DataFrame)
    or not bool(src_audit["source_available"].all())
):
    raise RuntimeError(
        "C6-A8-A3-S1 source lock must pass first."
    )

# Stable snapshot avoids globals()-mutation problems.
G = list(globals().items())

# ------------------------------------------------------------------
# 1. Resolve the already-built FD001 corruption contract
# ------------------------------------------------------------------
required_operator_keys = {
    "bias",
    "drift",
    "noise",
    "dropout",
    "stuck",
}

contract_hits = [
    (name, obj)
    for name, obj in G
    if str(name).startswith("FD001")
    and isinstance(obj, dict)
    and required_operator_keys.issubset(obj)
    and all(
        callable(obj[key])
        for key in required_operator_keys
    )
]

if len(contract_hits) != 1:
    raise RuntimeError(
        "Expected exactly one FD001 operator contract; "
        f"found {[name for name, _ in contract_hits]}."
    )

contract_name, contract = contract_hits[0]

# ------------------------------------------------------------------
# 2. Resolve canonical FD001 DEV corruption base
# ------------------------------------------------------------------
base_hits = [
    (name, obj)
    for name, obj in G
    if str(name).startswith("FD001")
    and isinstance(obj, pd.DataFrame)
    and obj.shape == (3852, 27)
    and {
        "unit_id",
        "cycle",
        "regime_id",
        "sensor_2",
    }.issubset(obj.columns)
]

if len(base_hits) == 0:
    raise RuntimeError(
        "Canonical FD001 DEV corruption base was not found."
    )

base_name, base = base_hits[0]

for other_name, other_base in base_hits[1:]:
    if not other_base.equals(base):
        raise RuntimeError(
            "Multiple non-identical FD001 canonical DEV bases "
            f"were found: {base_name!r} and {other_name!r}."
        )

base = base.copy(deep=True)

if (
    not isinstance(base.index, pd.RangeIndex)
    or not np.array_equal(
        base.index.to_numpy(),
        np.arange(len(base)),
    )
):
    raise RuntimeError(
        "Canonical DEV corruption base index must be 0..N-1."
    )

# ------------------------------------------------------------------
# 3. Resolve the frozen 315-scenario FD001 manifest
# ------------------------------------------------------------------
required_manifest_cols = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
}

manifest_hits = [
    (name, obj)
    for name, obj in G
    if str(name).startswith("FD001")
    and isinstance(obj, pd.DataFrame)
    and obj.shape == (315, 9)
    and set(obj.columns) == required_manifest_cols
]

if len(manifest_hits) == 0:
    raise RuntimeError(
        "FD001 315-scenario manifest was not found."
    )

manifest_name, manifest = manifest_hits[0]

manifest_ref = (
    manifest
    .sort_values("scenario_number")
    .reset_index(drop=True)
)

for other_name, other_manifest in manifest_hits[1:]:
    other_ref = (
        other_manifest
        .sort_values("scenario_number")
        .reset_index(drop=True)
    )

    if not other_ref.equals(manifest_ref):
        raise RuntimeError(
            "Multiple non-identical FD001 manifests were found: "
            f"{manifest_name!r} and {other_name!r}."
        )

# ------------------------------------------------------------------
# 4. FIT-only transferred references
# ------------------------------------------------------------------
if "sensor_scale" not in contract:
    raise RuntimeError(
        "FD001 contract is missing sensor_scale."
    )

if "dropout_mean_reference" not in contract:
    raise RuntimeError(
        "FD001 contract is missing dropout_mean_reference."
    )

sensor_scale = dict(
    contract["sensor_scale"]
)

dropout_mean = dict(
    contract["dropout_mean_reference"]
)

sensor = str(
    contract.get(
        "smoke_sensor",
        "sensor_2",
    )
)

if sensor not in base.columns:
    raise RuntimeError(
        f"Smoke sensor {sensor!r} is absent from DEV base."
    )

if sensor not in sensor_scale:
    raise RuntimeError(
        f"Smoke sensor {sensor!r} is absent from sensor-scale reference."
    )

# ------------------------------------------------------------------
# 5. Select one MEDIUM scenario per family
# ------------------------------------------------------------------
def row_for(family):

    q = manifest[
        (manifest["family"] == family)
        & (manifest["severity"] == "medium")
        & (manifest["sensor"] == sensor)
    ]

    if family in {"bias", "drift"}:
        q = q[
            q["direction"] == 1
        ]

    if len(q) != 1:
        raise RuntimeError(
            f"Non-unique smoke scenario for {family}: "
            f"{len(q)} rows."
        )

    return q.iloc[0]


rb = row_for("bias")
rd = row_for("drift")
rn = row_for("noise")
ro = row_for("dropout")
rs = row_for("stuck")

# ------------------------------------------------------------------
# 6. Execute exactly five transferred corruptions — COPIES ONLY
# ------------------------------------------------------------------
bias_out, bias_amount = contract["bias"](
    base.copy(deep=True),
    sensor,
    float(rb["alpha"]),
    int(rb["direction"]),
)

drift_out, drift_vec, final_drift = contract["drift"](
    base.copy(deep=True),
    sensor,
    float(rd["alpha"]),
    int(rd["direction"]),
)

noise_out, noise_vec, target_std = contract["noise"](
    base.copy(deep=True),
    sensor,
    float(rn["alpha"]),
    int(rn["seed"]),
)

drop_out, drop_mask, drop_imp = contract["dropout"](
    base.copy(deep=True),
    sensor,
    float(ro["requested_fraction"]),
    int(ro["seed"]),
)

stuck_out, stuck_mask, stuck_info = contract["stuck"](
    base.copy(deep=True),
    sensor,
    float(rs["requested_fraction"]),
)

x0 = base[
    sensor
].to_numpy(dtype=np.float64)

scale = float(
    sensor_scale[sensor]
)

# ------------------------------------------------------------------
# 7. Independent historical-semantics reconstruction
# ------------------------------------------------------------------

# ----- bias -----
bias_expected_amount = float(
    int(rb["direction"])
    * float(rb["alpha"])
    * scale
)

bias_ok = bool(
    np.isclose(
        bias_amount,
        bias_expected_amount,
        rtol=0.0,
        atol=1e-12,
    )
    and np.allclose(
        bias_out[sensor].to_numpy(dtype=np.float64),
        x0 + bias_expected_amount,
        rtol=0.0,
        atol=1e-12,
    )
)

# ----- drift -----
drift_final_expected = float(
    int(rd["direction"])
    * float(rd["alpha"])
    * scale
)

drift_expected = np.zeros(
    len(base),
    dtype=np.float64,
)

for _, group in base.groupby(
    "unit_id",
    sort=False,
):

    positions = (
        group
        .sort_values("cycle")
        .index
        .to_numpy(dtype=np.int64)
    )

    if len(positions) == 1:
        engine_drift = np.array(
            [drift_final_expected],
            dtype=np.float64,
        )
    else:
        engine_drift = np.linspace(
            0.0,
            drift_final_expected,
            len(positions),
            endpoint=True,
            dtype=np.float64,
        )

    drift_expected[
        positions
    ] = engine_drift

drift_ok = bool(
    np.isclose(
        final_drift,
        drift_final_expected,
        rtol=0.0,
        atol=1e-12,
    )
    and np.allclose(
        np.asarray(
            drift_vec,
            dtype=np.float64,
        ),
        drift_expected,
        rtol=0.0,
        atol=1e-12,
    )
    and np.allclose(
        drift_out[sensor].to_numpy(dtype=np.float64),
        x0 + drift_expected,
        rtol=0.0,
        atol=1e-12,
    )
)

# ----- noise -----
noise_std_expected = float(
    float(rn["alpha"])
    * scale
)

rng = np.random.default_rng(
    int(rn["seed"])
)

noise_expected = (
    rng.normal(
        0.0,
        1.0,
        len(base),
    )
    * noise_std_expected
)

noise_ok = bool(
    np.isclose(
        target_std,
        noise_std_expected,
        rtol=0.0,
        atol=1e-12,
    )
    and np.allclose(
        np.asarray(
            noise_vec,
            dtype=np.float64,
        ),
        noise_expected,
        rtol=0.0,
        atol=1e-12,
    )
    and np.allclose(
        noise_out[sensor].to_numpy(dtype=np.float64),
        x0 + noise_expected,
        rtol=0.0,
        atol=1e-12,
    )
)

# ----- dropout -----
rate = float(
    ro["requested_fraction"]
)

n_drop = int(
    round(
        rate
        * len(base)
    )
)

rng = np.random.default_rng(
    int(ro["seed"])
)

drop_positions = (
    rng.permutation(
        len(base)
    )[:n_drop]
)

drop_mask_expected = np.zeros(
    len(base),
    dtype=bool,
)

drop_mask_expected[
    drop_positions
] = True

drop_expected = x0.copy()

regime_values = (
    base["regime_id"]
    .to_numpy()
)

for position in drop_positions:

    key = (
        int(
            regime_values[
                position
            ]
        ),
        sensor,
    )

    if key not in dropout_mean:
        raise RuntimeError(
            f"Missing FD001 dropout reference for {key}."
        )

    drop_expected[
        position
    ] = float(
        dropout_mean[
            key
        ]
    )

drop_ok = bool(
    np.array_equal(
        np.asarray(
            drop_mask,
            dtype=bool,
        ),
        drop_mask_expected,
    )
    and np.allclose(
        drop_out[sensor].to_numpy(dtype=np.float64),
        drop_expected,
        rtol=0.0,
        atol=1e-12,
    )
)

# ----- stuck -----
fraction = float(
    rs["requested_fraction"]
)

stuck_expected = x0.copy()

stuck_mask_expected = np.zeros(
    len(base),
    dtype=bool,
)

for _, group in base.groupby(
    "unit_id",
    sort=False,
):

    group_sorted = (
        group
        .sort_values("cycle")
    )

    positions = (
        group_sorted
        .index
        .to_numpy(dtype=np.int64)
    )

    n_affected = int(
        round(
            fraction
            * len(positions)
        )
    )

    n_affected = max(
        1,
        n_affected,
    )

    n_affected = min(
        n_affected,
        len(positions) - 1,
    )

    onset = (
        len(positions)
        - n_affected
    )

    source_position = int(
        positions[
            onset - 1
        ]
    )

    affected_positions = (
        positions[
            onset:
        ]
    )

    freeze_value = float(
        base.loc[
            source_position,
            sensor,
        ]
    )

    stuck_expected[
        affected_positions
    ] = freeze_value

    stuck_mask_expected[
        affected_positions
    ] = True

stuck_ok = bool(
    np.array_equal(
        np.asarray(
            stuck_mask,
            dtype=bool,
        ),
        stuck_mask_expected,
    )
    and np.allclose(
        stuck_out[sensor].to_numpy(dtype=np.float64),
        stuck_expected,
        rtol=0.0,
        atol=1e-12,
    )
    and isinstance(
        stuck_info,
        pd.DataFrame,
    )
    and len(stuck_info)
    ==
    base["unit_id"].nunique()
)

# ------------------------------------------------------------------
# 8. Structural invariants
# ------------------------------------------------------------------
outputs = {
    "bias": bias_out,
    "drift": drift_out,
    "noise": noise_out,
    "dropout": drop_out,
    "stuck": stuck_out,
}

other_cols = [
    col
    for col in base.columns
    if col != sensor
]

shape_ok = all(
    output.shape
    ==
    base.shape

    for output
    in outputs.values()
)

columns_ok = all(
    list(output.columns)
    ==
    list(base.columns)

    for output
    in outputs.values()
)

non_target_ok = all(
    base[
        other_cols
    ].equals(
        output[
            other_cols
        ]
    )

    for output
    in outputs.values()
)

finite_ok = all(
    np.isfinite(
        output[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
    ).all()

    for output
    in outputs.values()
)

# Original canonical object must still equal our untouched copy.
canonical_unchanged = all(
    original.equals(
        base
    )

    for _, original
    in base_hits
)

# ------------------------------------------------------------------
# 9. Smoke summary
# ------------------------------------------------------------------
smoke_summary = pd.DataFrame(
    [
        [
            "bias",
            rb["scenario_id"],
            float(rb["alpha"]),
            len(base),
            bias_ok,
        ],
        [
            "drift",
            rd["scenario_id"],
            float(rd["alpha"]),
            int(
                np.count_nonzero(
                    drift_vec
                )
            ),
            drift_ok,
        ],
        [
            "noise",
            rn["scenario_id"],
            float(rn["alpha"]),
            len(base),
            noise_ok,
        ],
        [
            "dropout",
            ro["scenario_id"],
            rate,
            int(
                np.sum(
                    drop_mask
                )
            ),
            drop_ok,
        ],
        [
            "stuck",
            rs["scenario_id"],
            fraction,
            int(
                np.sum(
                    stuck_mask
                )
            ),
            stuck_ok,
        ],
    ],
    columns=[
        "family",
        "scenario_id",
        "medium_control",
        "affected_rows",
        "semantic_exact",
    ],
)

print("\n--- RESOLVED OBJECTS ---")
print("Operator contract :", contract_name)
print("DEV base          :", base_name)
print("Manifest          :", manifest_name)
print("Smoke sensor      :", sensor)

print("\n--- FIVE-FAMILY SMOKE RESULTS ---")
display(
    smoke_summary
)

# ------------------------------------------------------------------
# 10. Audit
# ------------------------------------------------------------------
audit = pd.DataFrame(
    [
        ["previous_step_passed", True],
        ["historical_source_lock_passed", True],
        ["operator_contract_unique", len(contract_hits) == 1],
        ["canonical_dev_base_resolved", len(base_hits) >= 1],
        ["manifest_resolved", len(manifest_hits) >= 1],

        ["bias_semantic_exact", bias_ok],
        ["drift_semantic_exact", drift_ok],
        ["noise_semantic_exact", noise_ok],
        ["dropout_semantic_exact", drop_ok],
        ["stuck_semantic_exact", stuck_ok],

        ["all_shapes_preserved", shape_ok],
        ["all_columns_preserved", columns_ok],
        ["all_non_target_columns_unchanged", non_target_ok],
        ["all_target_outputs_finite", finite_ok],
        ["canonical_dev_base_unchanged", canonical_unchanged],

        ["test_features_not_read", True],
        ["official_test_labels_not_read", True],
        ["test_not_scored", True],
        ["no_model_execution", True],
        ["no_model_retraining", True],
        ["no_threshold_modification", True],
        ["no_files_written", True],
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A4_AUDIT_628HC6A8A4 = audit

all_checks_passed_628hc6a8a4 = bool(
    audit[
        "passed"
    ].all()
)

print("\n--- C6-A8-A4 AUDIT ---")
display(
    audit
)

print("\n" + "=" * 110)
print("STEP 6.28H-C6-A8-A4 FINAL SUMMARY")
print("=" * 110)

print(
    "All five transferred operators semantically exact :",
    bool(
        smoke_summary[
            "semantic_exact"
        ].all()
    ),
)

print(
    "Canonical DEV base unchanged                      :",
    canonical_unchanged,
)

print(
    "FD001 TEST features read                           : False"
)

print(
    "Official FD001 TEST labels read                    : False"
)

print(
    "Model executed/retrained                           : False"
)

print(
    "Threshold modified                                 : False"
)

print(
    "Files written                                      : False"
)

print(
    "All C6-A8-A4 checks passed                         :",
    all_checks_passed_628hc6a8a4,
)

if not all_checks_passed_628hc6a8a4:

    raise RuntimeError(
        "C6-A8-A4 failed. "
        "Do not proceed to normalization/window-bridge execution."
    )

print(
    "\nPASS — five-family FD001 smoke corruption reproduces "
    "the recovered FD002 operator semantics exactly."
)

print(
    "Next: C6-A8-A5 — lock/smoke-test the FD001 normalization "
    "+ 30-cycle window bridge before the full 315-scenario run."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A4 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A4-R1 — FD001 FIVE-FAMILY CORRUPTION SMOKE / SEMANTIC AUDIT
# ==================================================================================

import numpy as np
import pandas as pd

print("=" * 112)
print("STEP 6.28H-C6-A8-A4-R1 — FD001 FIVE-FAMILY CORRUPTION SMOKE / SEMANTIC AUDIT")
print("=" * 112)

# ------------------------------------------------------------------------------
# 1. Stable notebook snapshot
# ------------------------------------------------------------------------------
G = list(globals().items())

REQUIRED_FAMILIES = {"bias", "drift", "noise", "dropout", "stuck"}

REQUIRED_MANIFEST_COLS = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
}

# ------------------------------------------------------------------------------
# 2. Resolve transferred FD001 operator contract
# ------------------------------------------------------------------------------
contract_candidates = []

for name, obj in G:
    if not isinstance(obj, dict):
        continue

    if not REQUIRED_FAMILIES.issubset(set(obj.keys())):
        continue

    if not all(callable(obj[k]) for k in REQUIRED_FAMILIES):
        continue

    score = 0
    name_upper = str(name).upper()

    if "FD001" in name_upper:
        score += 100

    if "A8A3" in name_upper:
        score += 20

    if "CORRUPTION" in name_upper:
        score += 10

    if "OPERATOR" in name_upper or "CONTRACT" in name_upper:
        score += 10

    contract_candidates.append(
        (score, str(name), obj)
    )

if len(contract_candidates) == 0:
    raise RuntimeError(
        "No five-family FD001 corruption operator contract found."
    )

contract_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

best_contract_score = contract_candidates[0][0]

best_contracts = [
    item
    for item in contract_candidates
    if item[0] == best_contract_score
]

if len(best_contracts) != 1:
    raise RuntimeError(
        "FD001 operator contract is ambiguous. Candidates: "
        + str([x[1] for x in best_contracts])
    )

_, contract_name, contract = best_contracts[0]

# ------------------------------------------------------------------------------
# 3. Resolve canonical FD001 DEV corruption base
# ------------------------------------------------------------------------------
base_candidates = []

for name, obj in G:
    if not isinstance(obj, pd.DataFrame):
        continue

    if obj.shape != (3852, 27):
        continue

    required_base_cols = {
        "unit_id",
        "cycle",
        "regime_id",
        "sensor_2",
    }

    if not required_base_cols.issubset(
        set(obj.columns)
    ):
        continue

    score = 0
    name_upper = str(name).upper()

    if "FD001" in name_upper:
        score += 100

    if "DEV" in name_upper:
        score += 20

    if "BASE" in name_upper:
        score += 20

    if "A8A3" in name_upper:
        score += 10

    base_candidates.append(
        (score, str(name), obj)
    )

if len(base_candidates) == 0:
    raise RuntimeError(
        "Canonical FD001 DEV corruption base (3852 x 27) not found."
    )

base_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

best_base_score = base_candidates[0][0]

best_bases = [
    item
    for item in base_candidates
    if item[0] == best_base_score
]

base_name = best_bases[0][1]
base_source = best_bases[0][2]

for _, other_name, other_obj in best_bases[1:]:
    if not other_obj.equals(base_source):
        raise RuntimeError(
            "Top-ranked FD001 DEV base candidates are not identical: "
            f"{base_name!r} vs {other_name!r}."
        )

base_before = base_source.copy(deep=True)
base = base_source.copy(deep=True)

if not np.array_equal(
    base.index.to_numpy(),
    np.arange(
        len(base),
        dtype=base.index.to_numpy().dtype,
    ),
):
    raise RuntimeError(
        "FD001 DEV base index must be canonical 0..3851."
    )

# ------------------------------------------------------------------------------
# 4. Resolve FD001 315-scenario manifest
# ------------------------------------------------------------------------------
manifest_candidates = []

for name, obj in G:
    if not isinstance(obj, pd.DataFrame):
        continue

    if obj.shape != (315, 9):
        continue

    if set(obj.columns) != REQUIRED_MANIFEST_COLS:
        continue

    score = 0
    name_upper = str(name).upper()

    if "FD001" in name_upper:
        score += 100

    if "A8A3" in name_upper:
        score += 20

    if "MANIFEST" in name_upper:
        score += 20

    manifest_candidates.append(
        (score, str(name), obj)
    )

if len(manifest_candidates) == 0:
    raise RuntimeError(
        "FD001 315-scenario manifest not found."
    )

manifest_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

best_manifest_score = manifest_candidates[0][0]

best_manifests = [
    item
    for item in manifest_candidates
    if item[0] == best_manifest_score
]

manifest_name = best_manifests[0][1]

manifest = (
    best_manifests[0][2]
    .copy(deep=True)
)

manifest_reference = (
    manifest
    .sort_values("scenario_number")
    .reset_index(drop=True)
)

for _, other_name, other_obj in best_manifests[1:]:

    other_reference = (
        other_obj
        .sort_values("scenario_number")
        .reset_index(drop=True)
    )

    if not other_reference.equals(
        manifest_reference
    ):
        raise RuntimeError(
            "Top-ranked FD001 manifest candidates are not identical: "
            f"{manifest_name!r} vs {other_name!r}."
        )

# ------------------------------------------------------------------------------
# 5. Resolve FD001 FIT-only corruption scale reference
# ------------------------------------------------------------------------------
scale_candidates = []

required_scale_cols = {
    "sensor",
    "pooled_within_regime_std",
    "global_raw_std",
    "global_to_within_ratio",
}

for name, obj in G:
    if not isinstance(obj, pd.DataFrame):
        continue

    if obj.shape != (15, 4):
        continue

    if set(obj.columns) != required_scale_cols:
        continue

    score = 0
    name_upper = str(name).upper()

    if "FD001" in name_upper:
        score += 100

    if "A8A2" in name_upper:
        score += 20

    if "SCALE" in name_upper:
        score += 20

    scale_candidates.append(
        (score, str(name), obj)
    )

if len(scale_candidates) == 0:
    raise RuntimeError(
        "FD001 FIT-only 15-row scale reference not found."
    )

scale_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

scale_name = scale_candidates[0][1]

scale_df = (
    scale_candidates[0][2]
    .copy(deep=True)
)

sensor_scale = {
    str(row["sensor"]):
        float(
            row["pooled_within_regime_std"]
        )
    for _, row in scale_df.iterrows()
}

# ------------------------------------------------------------------------------
# 6. Resolve FD001 FIT-only dropout-imputation reference
# ------------------------------------------------------------------------------
dropref_candidates = []

required_dropref_cols = {
    "regime_id",
    "sensor",
    "mean",
}

for name, obj in G:
    if not isinstance(obj, pd.DataFrame):
        continue

    if obj.shape != (15, 3):
        continue

    if set(obj.columns) != required_dropref_cols:
        continue

    score = 0
    name_upper = str(name).upper()

    if "FD001" in name_upper:
        score += 100

    if "A8A2" in name_upper:
        score += 20

    if "DROPOUT" in name_upper:
        score += 20

    dropref_candidates.append(
        (score, str(name), obj)
    )

if len(dropref_candidates) == 0:
    raise RuntimeError(
        "FD001 FIT-only dropout-imputation reference not found."
    )

dropref_candidates.sort(
    key=lambda x: (-x[0], x[1])
)

dropref_name = dropref_candidates[0][1]

dropref_df = (
    dropref_candidates[0][2]
    .copy(deep=True)
)

dropout_mean = {
    (
        int(row["regime_id"]),
        str(row["sensor"]),
    ):
        float(row["mean"])

    for _, row
    in dropref_df.iterrows()
}

# ------------------------------------------------------------------------------
# 7. Select one MEDIUM scenario per family on sensor_2
# ------------------------------------------------------------------------------
sensor = "sensor_2"

if sensor not in base.columns:
    raise RuntimeError(
        "sensor_2 absent from FD001 DEV base."
    )

if sensor not in sensor_scale:
    raise RuntimeError(
        "sensor_2 absent from FD001 scale reference."
    )

def select_smoke_row(family):

    q = manifest[
        (
            manifest["family"]
            .astype(str)
            ==
            family
        )
        &
        (
            manifest["severity"]
            .astype(str)
            ==
            "medium"
        )
        &
        (
            manifest["sensor"]
            .astype(str)
            ==
            sensor
        )
    ].copy()

    if family in {
        "bias",
        "drift",
    }:

        q = q[
            q["direction"]
            .astype(float)
            ==
            1.0
        ]

    if len(q) != 1:
        raise RuntimeError(
            f"Expected exactly one medium smoke row "
            f"for family={family!r}, sensor={sensor!r}; "
            f"found {len(q)}."
        )

    return q.iloc[0]


row_bias = select_smoke_row(
    "bias"
)

row_drift = select_smoke_row(
    "drift"
)

row_noise = select_smoke_row(
    "noise"
)

row_dropout = select_smoke_row(
    "dropout"
)

row_stuck = select_smoke_row(
    "stuck"
)

# ------------------------------------------------------------------------------
# 8. Execute exactly five smoke corruptions on COPIES only
# ------------------------------------------------------------------------------
bias_out, bias_amount = (
    contract["bias"](
        base.copy(deep=True),
        sensor,
        float(
            row_bias["alpha"]
        ),
        int(
            row_bias["direction"]
        ),
    )
)

drift_out, drift_vector, final_drift = (
    contract["drift"](
        base.copy(deep=True),
        sensor,
        float(
            row_drift["alpha"]
        ),
        int(
            row_drift["direction"]
        ),
    )
)

noise_out, noise_vector, target_std = (
    contract["noise"](
        base.copy(deep=True),
        sensor,
        float(
            row_noise["alpha"]
        ),
        int(
            row_noise["seed"]
        ),
    )
)

dropout_out, dropout_mask, dropout_imputation_values = (
    contract["dropout"](
        base.copy(deep=True),
        sensor,
        float(
            row_dropout[
                "requested_fraction"
            ]
        ),
        int(
            row_dropout["seed"]
        ),
    )
)

stuck_out, stuck_mask, stuck_info = (
    contract["stuck"](
        base.copy(deep=True),
        sensor,
        float(
            row_stuck[
                "requested_fraction"
            ]
        ),
    )
)

x_original = (
    base[sensor]
    .to_numpy(
        dtype=np.float64
    )
)

scale = float(
    sensor_scale[
        sensor
    ]
)

# ------------------------------------------------------------------------------
# 9. Independent reconstruction — BIAS
# ------------------------------------------------------------------------------
expected_bias_amount = float(
    int(
        row_bias["direction"]
    )
    *
    float(
        row_bias["alpha"]
    )
    *
    scale
)

bias_semantic_exact = bool(
    np.isclose(
        float(
            bias_amount
        ),
        expected_bias_amount,
        rtol=0.0,
        atol=1e-12,
    )
    and
    np.allclose(
        bias_out[
            sensor
        ].to_numpy(
            dtype=np.float64
        ),
        x_original
        +
        expected_bias_amount,
        rtol=0.0,
        atol=1e-12,
    )
)

# ------------------------------------------------------------------------------
# 10. Independent reconstruction — DRIFT
# ------------------------------------------------------------------------------
expected_final_drift = float(
    int(
        row_drift["direction"]
    )
    *
    float(
        row_drift["alpha"]
    )
    *
    scale
)

expected_drift_vector = (
    np.zeros(
        len(base),
        dtype=np.float64,
    )
)

for _, group in (
    base.groupby(
        "unit_id",
        sort=False,
    )
):

    positions = (
        group
        .sort_values(
            "cycle"
        )
        .index
        .to_numpy(
            dtype=np.int64
        )
    )

    n_rows = len(
        positions
    )

    if n_rows == 1:

        engine_drift = (
            np.array(
                [
                    expected_final_drift
                ],
                dtype=np.float64,
            )
        )

    else:

        engine_drift = (
            np.linspace(
                0.0,
                expected_final_drift,
                num=n_rows,
                endpoint=True,
                dtype=np.float64,
            )
        )

    expected_drift_vector[
        positions
    ] = engine_drift

drift_semantic_exact = bool(
    np.isclose(
        float(
            final_drift
        ),
        expected_final_drift,
        rtol=0.0,
        atol=1e-12,
    )
    and
    np.allclose(
        np.asarray(
            drift_vector,
            dtype=np.float64,
        ),
        expected_drift_vector,
        rtol=0.0,
        atol=1e-12,
    )
    and
    np.allclose(
        drift_out[
            sensor
        ].to_numpy(
            dtype=np.float64
        ),
        x_original
        +
        expected_drift_vector,
        rtol=0.0,
        atol=1e-12,
    )
)

# ------------------------------------------------------------------------------
# 11. Independent reconstruction — NOISE
# ------------------------------------------------------------------------------
expected_target_std = float(
    float(
        row_noise["alpha"]
    )
    *
    scale
)

rng_noise = (
    np.random.default_rng(
        int(
            row_noise["seed"]
        )
    )
)

expected_noise_vector = (
    rng_noise.normal(
        loc=0.0,
        scale=1.0,
        size=len(base),
    )
    *
    expected_target_std
)

noise_semantic_exact = bool(
    np.isclose(
        float(
            target_std
        ),
        expected_target_std,
        rtol=0.0,
        atol=1e-12,
    )
    and
    np.allclose(
        np.asarray(
            noise_vector,
            dtype=np.float64,
        ),
        expected_noise_vector,
        rtol=0.0,
        atol=1e-12,
    )
    and
    np.allclose(
        noise_out[
            sensor
        ].to_numpy(
            dtype=np.float64
        ),
        x_original
        +
        expected_noise_vector,
        rtol=0.0,
        atol=1e-12,
    )
)

# ------------------------------------------------------------------------------
# 12. Independent reconstruction — DROPOUT
# ------------------------------------------------------------------------------
dropout_rate = float(
    row_dropout[
        "requested_fraction"
    ]
)

n_drop = int(
    round(
        dropout_rate
        *
        len(base)
    )
)

rng_dropout = (
    np.random.default_rng(
        int(
            row_dropout["seed"]
        )
    )
)

expected_drop_positions = (
    rng_dropout.permutation(
        len(base)
    )[
        :n_drop
    ]
)

expected_dropout_mask = (
    np.zeros(
        len(base),
        dtype=bool,
    )
)

expected_dropout_mask[
    expected_drop_positions
] = True

expected_dropout_values = (
    x_original.copy()
)

regime_values = (
    base[
        "regime_id"
    ].to_numpy()
)

for position in (
    expected_drop_positions
):

    key = (
        int(
            regime_values[
                position
            ]
        ),
        sensor,
    )

    if key not in dropout_mean:
        raise RuntimeError(
            "Missing FD001 FIT-only dropout mean "
            f"for key={key!r}."
        )

    expected_dropout_values[
        position
    ] = float(
        dropout_mean[
            key
        ]
    )

dropout_semantic_exact = bool(
    np.array_equal(
        np.asarray(
            dropout_mask,
            dtype=bool,
        ),
        expected_dropout_mask,
    )
    and
    np.allclose(
        dropout_out[
            sensor
        ].to_numpy(
            dtype=np.float64
        ),
        expected_dropout_values,
        rtol=0.0,
        atol=1e-12,
    )
)

# ------------------------------------------------------------------------------
# 13. Independent reconstruction — STUCK
# ------------------------------------------------------------------------------
affected_fraction = float(
    row_stuck[
        "requested_fraction"
    ]
)

expected_stuck_values = (
    x_original.copy()
)

expected_stuck_mask = (
    np.zeros(
        len(base),
        dtype=bool,
    )
)

expected_stuck_records = []

for unit_id, group in (
    base.groupby(
        "unit_id",
        sort=False,
    )
):

    group_sorted = (
        group
        .sort_values(
            "cycle"
        )
    )

    positions = (
        group_sorted
        .index
        .to_numpy(
            dtype=np.int64
        )
    )

    n_rows = len(
        positions
    )

    if n_rows < 2:
        raise RuntimeError(
            "Stuck smoke audit encountered "
            "an engine with fewer than two cycles."
        )

    n_affected = int(
        round(
            affected_fraction
            *
            n_rows
        )
    )

    n_affected = max(
        1,
        n_affected,
    )

    n_affected = min(
        n_affected,
        n_rows - 1,
    )

    onset_position = (
        n_rows
        -
        n_affected
    )

    source_position = int(
        positions[
            onset_position - 1
        ]
    )

    affected_positions = (
        positions[
            onset_position:
        ]
    )

    freeze_value = float(
        base.loc[
            source_position,
            sensor,
        ]
    )

    expected_stuck_values[
        affected_positions
    ] = freeze_value

    expected_stuck_mask[
        affected_positions
    ] = True

    expected_stuck_records.append(
        {
            "unit_id":
                unit_id,

            "trajectory_length":
                n_rows,

            "requested_fraction":
                affected_fraction,

            "affected_count":
                n_affected,

            "actual_fraction":
                n_affected / n_rows,

            "source_cycle":
                float(
                    group_sorted.iloc[
                        onset_position - 1
                    ][
                        "cycle"
                    ]
                ),

            "stuck_start_cycle":
                float(
                    group_sorted.iloc[
                        onset_position
                    ][
                        "cycle"
                    ]
                ),

            "freeze_value":
                freeze_value,
        }
    )

expected_stuck_info = (
    pd.DataFrame(
        expected_stuck_records
    )
)

stuck_semantic_exact = bool(
    np.array_equal(
        np.asarray(
            stuck_mask,
            dtype=bool,
        ),
        expected_stuck_mask,
    )
    and
    np.allclose(
        stuck_out[
            sensor
        ].to_numpy(
            dtype=np.float64
        ),
        expected_stuck_values,
        rtol=0.0,
        atol=1e-12,
    )
    and
    isinstance(
        stuck_info,
        pd.DataFrame,
    )
    and
    list(
        stuck_info.columns
    )
    ==
    list(
        expected_stuck_info.columns
    )
    and
    stuck_info.reset_index(
        drop=True
    ).equals(
        expected_stuck_info.reset_index(
            drop=True
        )
    )
)

# ------------------------------------------------------------------------------
# 14. Structural invariants
# ------------------------------------------------------------------------------
outputs = {
    "bias":
        bias_out,

    "drift":
        drift_out,

    "noise":
        noise_out,

    "dropout":
        dropout_out,

    "stuck":
        stuck_out,
}

other_columns = [
    column
    for column in base.columns
    if column != sensor
]

all_shapes_preserved = bool(
    all(
        output.shape
        ==
        base.shape

        for output
        in outputs.values()
    )
)

all_columns_preserved = bool(
    all(
        list(
            output.columns
        )
        ==
        list(
            base.columns
        )

        for output
        in outputs.values()
    )
)

all_non_target_columns_unchanged = bool(
    all(
        output[
            other_columns
        ].equals(
            base[
                other_columns
            ]
        )

        for output
        in outputs.values()
    )
)

all_target_outputs_finite = bool(
    all(
        np.isfinite(
            output[
                sensor
            ].to_numpy(
                dtype=np.float64
            )
        ).all()

        for output
        in outputs.values()
    )
)

canonical_dev_base_unchanged = bool(
    base_source.equals(
        base_before
    )
)

# ------------------------------------------------------------------------------
# 15. Smoke summary
# ------------------------------------------------------------------------------
smoke_summary = pd.DataFrame(
    [
        {
            "family":
                "bias",

            "scenario_id":
                row_bias[
                    "scenario_id"
                ],

            "severity":
                row_bias[
                    "severity"
                ],

            "control_value":
                float(
                    row_bias[
                        "alpha"
                    ]
                ),

            "affected_rows":
                len(base),

            "semantic_exact":
                bias_semantic_exact,
        },

        {
            "family":
                "drift",

            "scenario_id":
                row_drift[
                    "scenario_id"
                ],

            "severity":
                row_drift[
                    "severity"
                ],

            "control_value":
                float(
                    row_drift[
                        "alpha"
                    ]
                ),

            "affected_rows":
                int(
                    np.count_nonzero(
                        np.asarray(
                            drift_vector,
                            dtype=np.float64,
                        )
                    )
                ),

            "semantic_exact":
                drift_semantic_exact,
        },

        {
            "family":
                "noise",

            "scenario_id":
                row_noise[
                    "scenario_id"
                ],

            "severity":
                row_noise[
                    "severity"
                ],

            "control_value":
                float(
                    row_noise[
                        "alpha"
                    ]
                ),

            "affected_rows":
                len(base),

            "semantic_exact":
                noise_semantic_exact,
        },

        {
            "family":
                "dropout",

            "scenario_id":
                row_dropout[
                    "scenario_id"
                ],

            "severity":
                row_dropout[
                    "severity"
                ],

            "control_value":
                dropout_rate,

            "affected_rows":
                int(
                    np.sum(
                        np.asarray(
                            dropout_mask,
                            dtype=bool,
                        )
                    )
                ),

            "semantic_exact":
                dropout_semantic_exact,
        },

        {
            "family":
                "stuck",

            "scenario_id":
                row_stuck[
                    "scenario_id"
                ],

            "severity":
                row_stuck[
                    "severity"
                ],

            "control_value":
                affected_fraction,

            "affected_rows":
                int(
                    np.sum(
                        np.asarray(
                            stuck_mask,
                            dtype=bool,
                        )
                    )
                ),

            "semantic_exact":
                stuck_semantic_exact,
        },
    ]
)

print("\n--- RESOLVED OBJECTS ---")
print(
    "Operator contract       :",
    contract_name,
)
print(
    "Canonical DEV base      :",
    base_name,
)
print(
    "315-scenario manifest   :",
    manifest_name,
)
print(
    "FIT scale reference     :",
    scale_name,
)
print(
    "FIT dropout reference   :",
    dropref_name,
)
print(
    "Smoke sensor            :",
    sensor,
)

print(
    "\n--- FIVE-FAMILY SMOKE RESULTS ---"
)

display(
    smoke_summary
)

# ------------------------------------------------------------------------------
# 16. Audit
# ------------------------------------------------------------------------------
audit = pd.DataFrame(
    [
        [
            "operator_contract_resolved",
            True,
        ],
        [
            "canonical_dev_base_resolved",
            True,
        ],
        [
            "manifest_resolved",
            True,
        ],
        [
            "fit_scale_reference_resolved",
            True,
        ],
        [
            "fit_dropout_reference_resolved",
            True,
        ],
        [
            "bias_semantic_exact",
            bias_semantic_exact,
        ],
        [
            "drift_semantic_exact",
            drift_semantic_exact,
        ],
        [
            "noise_semantic_exact",
            noise_semantic_exact,
        ],
        [
            "dropout_semantic_exact",
            dropout_semantic_exact,
        ],
        [
            "stuck_semantic_exact",
            stuck_semantic_exact,
        ],
        [
            "all_shapes_preserved",
            all_shapes_preserved,
        ],
        [
            "all_columns_preserved",
            all_columns_preserved,
        ],
        [
            "all_non_target_columns_unchanged",
            all_non_target_columns_unchanged,
        ],
        [
            "all_target_outputs_finite",
            all_target_outputs_finite,
        ],
        [
            "canonical_dev_base_unchanged",
            canonical_dev_base_unchanged,
        ],
        [
            "test_features_not_read",
            True,
        ],
        [
            "official_test_labels_not_read",
            True,
        ],
        [
            "test_not_scored",
            True,
        ],
        [
            "no_model_execution",
            True,
        ],
        [
            "no_model_retraining",
            True,
        ],
        [
            "no_mc_dropout_execution",
            True,
        ],
        [
            "no_threshold_modification",
            True,
        ],
        [
            "no_files_written",
            True,
        ],
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A4_R1_AUDIT_628HC6A8A4R1 = (
    audit
)

all_checks_passed_628hc6a8a4r1 = bool(
    audit[
        "passed"
    ].all()
)

print(
    "\n--- C6-A8-A4-R1 AUDIT ---"
)

display(
    audit
)

print(
    "\n"
    +
    "=" * 112
)

print(
    "STEP 6.28H-C6-A8-A4-R1 FINAL SUMMARY"
)

print(
    "=" * 112
)

print(
    "All five corruption semantics exact :",
    bool(
        smoke_summary[
            "semantic_exact"
        ].all()
    ),
)

print(
    "Canonical DEV base unchanged        :",
    canonical_dev_base_unchanged,
)

print(
    "FD001 TEST features read             : False"
)

print(
    "Official FD001 TEST labels read      : False"
)

print(
    "Model executed/retrained             : False"
)

print(
    "MC dropout executed                  : False"
)

print(
    "Threshold modified                   : False"
)

print(
    "Files written                        : False"
)

print(
    "All C6-A8-A4-R1 checks passed        :",
    all_checks_passed_628hc6a8a4r1,
)

if not all_checks_passed_628hc6a8a4r1:

    raise RuntimeError(
        "C6-A8-A4-R1 failed. "
        "Do not proceed to normalization/window bridge."
    )

print(
    "\nPASS — FD001 five-family smoke corruption reproduces "
    "the recovered historical semantics exactly."
)

print(
    "Next: C6-A8-A5 — lock and smoke-test the FD001 normalization "
    "+ 30-cycle window bridge before the full 315-scenario run."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A4-R1 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A5 — FD001 NORMALIZATION + 30-CYCLE WINDOW BRIDGE LOCK
# ==================================================================================

import numpy as np
import pandas as pd

print("=" * 110)
print("STEP 6.28H-C6-A8-A5 — FD001 NORMALIZATION + 30-CYCLE WINDOW BRIDGE LOCK")
print("=" * 110)

# ---- 1. Require A4-R1 PASS --------------------------------------------------------
a4 = C6_A8_A4_R1_AUDIT_628HC6A8A4R1

if not isinstance(a4, pd.DataFrame) or not bool(a4["passed"].all()):
    raise RuntimeError("C6-A8-A4-R1 must pass before C6-A8-A5.")

# ---- 2. Exact locked objects -------------------------------------------------------
fit_raw = FD001_FIT_RAW_628HC6A3R1.copy(deep=True)
dev_base = FD001_DEV_CORRUPTION_BASE_628HC6A8A3.copy(deep=True)

features = list(FD001_MODEL_FEATURES_628HC6A4)

manifest = FD001_CORRUPTION_MANIFEST_628HC6A8A3.copy(deep=True)

ops = FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3

X_dev_locked = np.asarray(
    FD001_X_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float64,
)

meta_dev_locked = (
    FD001_DEV_WINDOW_METADATA_628HC6A5R1
    .copy(deep=True)
    .reset_index(drop=True)
)

if fit_raw.shape != (16779, 26):
    raise RuntimeError(
        f"Unexpected FIT raw shape: {fit_raw.shape}"
    )

if dev_base.shape != (3852, 27):
    raise RuntimeError(
        f"Unexpected DEV corruption-base shape: {dev_base.shape}"
    )

if len(features) != 17:
    raise RuntimeError(
        f"Expected 17 model features, got {len(features)}"
    )

if X_dev_locked.shape != (3272, 30, 17):
    raise RuntimeError(
        f"Unexpected locked DEV-window shape: {X_dev_locked.shape}"
    )

required_dev = {
    "unit_id",
    "cycle",
    "regime_id",
    *features,
}

missing_dev = sorted(
    required_dev.difference(
        dev_base.columns
    )
)

missing_fit = sorted(
    set(features).difference(
        fit_raw.columns
    )
)

if missing_dev:
    raise RuntimeError(
        f"DEV base missing columns: {missing_dev}"
    )

if missing_fit:
    raise RuntimeError(
        f"FIT raw missing model features: {missing_fit}"
    )

# ---- 3. FIT-only normalization contract -------------------------------------------
# C6-A4 used population std (ddof=0).

fit_mean = (
    fit_raw[features]
    .mean(axis=0)
    .astype(np.float64)
)

fit_std = (
    fit_raw[features]
    .std(axis=0, ddof=0)
    .astype(np.float64)
)

if not np.isfinite(
    fit_mean.to_numpy()
).all():
    raise RuntimeError(
        "Non-finite FIT means."
    )

if not np.isfinite(
    fit_std.to_numpy()
).all():
    raise RuntimeError(
        "Non-finite FIT std values."
    )

if not np.all(
    fit_std.to_numpy() > 0.0
):
    raise RuntimeError(
        "All retained features must have positive FIT std."
    )


def normalize_fd001_628hc6a8a5(raw_df):

    out = raw_df.copy(deep=True)

    out.loc[:, features] = (
        out[features].astype(np.float64)
        -
        fit_mean
    ) / fit_std

    return out


def build_windows_fd001_628hc6a8a5(normalized_df):

    windows = []
    rows = []

    for unit_id, g in normalized_df.groupby(
        "unit_id",
        sort=False,
    ):

        g = g.sort_values(
            "cycle",
            kind="mergesort",
        )

        values = (
            g[features]
            .to_numpy(
                dtype=np.float64
            )
        )

        cycles = (
            g["cycle"]
            .to_numpy()
        )

        positions = (
            g.index
            .to_numpy(
                dtype=np.int64
            )
        )

        for start in range(
            len(g) - 30 + 1
        ):

            stop = (
                start
                +
                30
            )

            windows.append(
                values[
                    start:stop
                ]
            )

            rows.append(
                {
                    "engine_id":
                        unit_id,

                    "start_cycle":
                        cycles[
                            start
                        ],

                    "end_cycle":
                        cycles[
                            stop - 1
                        ],

                    "endpoint_row_position":
                        positions[
                            stop - 1
                        ],
                }
            )

    return (
        np.asarray(
            windows,
            dtype=np.float64,
        ),
        pd.DataFrame(
            rows,
            columns=[
                "engine_id",
                "start_cycle",
                "end_cycle",
                "endpoint_row_position",
            ],
        ),
    )

# ---- 4. CLEAN DEV identity against frozen C6-A5-R1 -------------------------------

clean_norm = (
    normalize_fd001_628hc6a8a5(
        dev_base
    )
)

X_clean, meta_clean = (
    build_windows_fd001_628hc6a8a5(
        clean_norm
    )
)

if X_clean.shape != (
    3272,
    30,
    17,
):
    raise RuntimeError(
        "Rebuilt clean window shape mismatch: "
        f"{X_clean.shape}"
    )

clean_max_err = float(
    np.max(
        np.abs(
            X_clean
            -
            X_dev_locked
        )
    )
)

clean_exact = bool(
    np.array_equal(
        X_clean,
        X_dev_locked,
    )
)

clean_close = bool(
    np.allclose(
        X_clean,
        X_dev_locked,
        rtol=0.0,
        atol=1e-12,
    )
)

meta_exact = bool(
    meta_clean.equals(
        meta_dev_locked
    )
)

if not clean_close:
    raise RuntimeError(
        "Clean normalization/window bridge does not reproduce "
        "locked DEV windows. "
        f"max_abs_error={clean_max_err:.18e}"
    )

if not meta_exact:
    raise RuntimeError(
        "Clean window metadata does not match "
        "C6-A5-R1 exactly."
    )

# ---- 5. Select medium sensor_2 smoke cases ----------------------------------------

SENSOR = "sensor_2"


def smoke_row(family):

    q = manifest[
        (
            manifest["family"].astype(str)
            ==
            family
        )
        &
        (
            manifest["severity"].astype(str)
            ==
            "medium"
        )
        &
        (
            manifest["sensor"].astype(str)
            ==
            SENSOR
        )
    ].copy()

    if family in {
        "bias",
        "drift",
    }:

        q = q[
            q["direction"].astype(float)
            ==
            1.0
        ]

    if len(q) != 1:

        raise RuntimeError(
            f"Expected exactly one medium "
            f"{family}/{SENSOR} scenario; "
            f"found {len(q)}."
        )

    return q.iloc[0]


r_bias = smoke_row(
    "bias"
)

r_drift = smoke_row(
    "drift"
)

r_noise = smoke_row(
    "noise"
)

r_dropout = smoke_row(
    "dropout"
)

r_stuck = smoke_row(
    "stuck"
)

# ---- 6. Five fresh smoke corruptions on COPIES ------------------------------------

smoke_raw = {

    "bias":
        ops["bias"](
            dev_base.copy(
                deep=True
            ),
            SENSOR,
            float(
                r_bias["alpha"]
            ),
            int(
                r_bias["direction"]
            ),
        )[0],

    "drift":
        ops["drift"](
            dev_base.copy(
                deep=True
            ),
            SENSOR,
            float(
                r_drift["alpha"]
            ),
            int(
                r_drift["direction"]
            ),
        )[0],

    "noise":
        ops["noise"](
            dev_base.copy(
                deep=True
            ),
            SENSOR,
            float(
                r_noise["alpha"]
            ),
            int(
                r_noise["seed"]
            ),
        )[0],

    "dropout":
        ops["dropout"](
            dev_base.copy(
                deep=True
            ),
            SENSOR,
            float(
                r_dropout[
                    "requested_fraction"
                ]
            ),
            int(
                r_dropout["seed"]
            ),
        )[0],

    "stuck":
        ops["stuck"](
            dev_base.copy(
                deep=True
            ),
            SENSOR,
            float(
                r_stuck[
                    "requested_fraction"
                ]
            ),
        )[0],
}

# ---- 7. Normalization/window smoke -------------------------------------------------

sensor_idx = (
    features.index(
        SENSOR
    )
)

other_idx = [
    i
    for i in range(17)
    if i != sensor_idx
]

summary_rows = []

for family in [
    "bias",
    "drift",
    "noise",
    "dropout",
    "stuck",
]:

    raw_c = (
        smoke_raw[
            family
        ]
    )

    norm_c = (
        normalize_fd001_628hc6a8a5(
            raw_c
        )
    )

    X_c, meta_c = (
        build_windows_fd001_628hc6a8a5(
            norm_c
        )
    )

    non_target_diff = float(
        np.max(
            np.abs(
                X_c[
                    :,
                    :,
                    other_idx,
                ]
                -
                X_dev_locked[
                    :,
                    :,
                    other_idx,
                ]
            )
        )
    )

    target_diff = np.abs(
        X_c[
            :,
            :,
            sensor_idx,
        ]
        -
        X_dev_locked[
            :,
            :,
            sensor_idx,
        ]
    )

    raw_delta = (
        raw_c[
            SENSOR
        ].to_numpy(
            dtype=np.float64
        )
        -
        dev_base[
            SENSOR
        ].to_numpy(
            dtype=np.float64
        )
    )

    norm_delta = (
        norm_c[
            SENSOR
        ].to_numpy(
            dtype=np.float64
        )
        -
        clean_norm[
            SENSOR
        ].to_numpy(
            dtype=np.float64
        )
    )

    expected_norm_delta = (
        raw_delta
        /
        float(
            fit_std.loc[
                SENSOR
            ]
        )
    )

    summary_rows.append(
        {
            "family":
                family,

            "shape_exact":
                X_c.shape
                ==
                (
                    3272,
                    30,
                    17,
                ),

            "finite":
                bool(
                    np.isfinite(
                        X_c
                    ).all()
                ),

            "metadata_exact":
                bool(
                    meta_c.equals(
                        meta_dev_locked
                    )
                ),

            "non_target_exact":
                bool(
                    non_target_diff
                    ==
                    0.0
                ),

            "target_changed":
                bool(
                    np.count_nonzero(
                        target_diff
                        >
                        1e-12
                    )
                    >
                    0
                ),

            "normalization_delta_exact":
                bool(
                    np.allclose(
                        norm_delta,
                        expected_norm_delta,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),

            "changed_target_window_entries":
                int(
                    np.count_nonzero(
                        target_diff
                        >
                        1e-12
                    )
                ),

            "non_target_max_abs_diff":
                non_target_diff,
        }
    )

bridge_summary = pd.DataFrame(
    summary_rows
)

print(
    "\n--- CLEAN BRIDGE IDENTITY ---"
)

print(
    "Locked DEV windows :",
    X_dev_locked.shape,
)

print(
    "Rebuilt DEV windows:",
    X_clean.shape,
)

print(
    "Array exact        :",
    clean_exact,
)

print(
    "Within 1e-12       :",
    clean_close,
)

print(
    "Max abs error      :",
    clean_max_err,
)

print(
    "Metadata exact     :",
    meta_exact,
)

print(
    "\n--- FIVE-FAMILY NORMALIZATION/WINDOW SMOKE ---"
)

display(
    bridge_summary
)

# ---- 8. Audit ---------------------------------------------------------------------

audit = pd.DataFrame(
    [
        [
            "previous_step_passed",
            True,
        ],
        [
            "fit_reference_source_fit_only",
            True,
        ],
        [
            "normalization_ddof0",
            True,
        ],
        [
            "clean_window_shape_3272x30x17",
            X_clean.shape
            ==
            (
                3272,
                30,
                17,
            ),
        ],
        [
            "clean_windows_within_1e12",
            clean_close,
        ],
        [
            "clean_metadata_exact",
            meta_exact,
        ],
        [
            "all_smoke_shapes_exact",
            bool(
                bridge_summary[
                    "shape_exact"
                ].all()
            ),
        ],
        [
            "all_smoke_finite",
            bool(
                bridge_summary[
                    "finite"
                ].all()
            ),
        ],
        [
            "all_smoke_metadata_exact",
            bool(
                bridge_summary[
                    "metadata_exact"
                ].all()
            ),
        ],
        [
            "all_smoke_non_target_exact",
            bool(
                bridge_summary[
                    "non_target_exact"
                ].all()
            ),
        ],
        [
            "all_smoke_target_changed",
            bool(
                bridge_summary[
                    "target_changed"
                ].all()
            ),
        ],
        [
            "all_smoke_normalization_delta_exact",
            bool(
                bridge_summary[
                    "normalization_delta_exact"
                ].all()
            ),
        ],
        [
            "canonical_dev_base_unchanged",
            bool(
                FD001_DEV_CORRUPTION_BASE_628HC6A8A3.equals(
                    dev_base
                )
            ),
        ],
        [
            "test_features_not_read",
            True,
        ],
        [
            "official_test_labels_not_read",
            True,
        ],
        [
            "test_not_scored",
            True,
        ],
        [
            "no_model_execution",
            True,
        ],
        [
            "no_model_retraining",
            True,
        ],
        [
            "no_mc_dropout_execution",
            True,
        ],
        [
            "no_threshold_modification",
            True,
        ],
        [
            "no_files_written",
            True,
        ],
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A5_AUDIT_628HC6A8A5 = (
    audit
)

all_passed = bool(
    audit[
        "passed"
    ].all()
)

print(
    "\n--- C6-A8-A5 AUDIT ---"
)

display(
    audit
)

print(
    "\n"
    +
    "=" * 110
)

print(
    "STEP 6.28H-C6-A8-A5 FINAL SUMMARY"
)

print(
    "=" * 110
)

print(
    "Clean bridge within 1e-12          :",
    clean_close,
)

print(
    "Clean metadata exact               :",
    meta_exact,
)

print(
    "Five smoke shapes exact            :",
    bool(
        bridge_summary[
            "shape_exact"
        ].all()
    ),
)

print(
    "Five smoke outputs finite          :",
    bool(
        bridge_summary[
            "finite"
        ].all()
    ),
)

print(
    "Non-target channels exact          :",
    bool(
        bridge_summary[
            "non_target_exact"
        ].all()
    ),
)

print(
    "Target channel changed in all five :",
    bool(
        bridge_summary[
            "target_changed"
        ].all()
    ),
)

print(
    "Normalization delta identity       :",
    bool(
        bridge_summary[
            "normalization_delta_exact"
        ].all()
    ),
)

print(
    "Canonical DEV base unchanged       :",
    bool(
        FD001_DEV_CORRUPTION_BASE_628HC6A8A3.equals(
            dev_base
        )
    ),
)

print(
    "FD001 TEST features read            : False"
)

print(
    "Official FD001 TEST labels read     : False"
)

print(
    "Model executed/retrained            : False"
)

print(
    "MC dropout executed                 : False"
)

print(
    "Threshold modified                  : False"
)

print(
    "Files written                       : False"
)

print(
    "All C6-A8-A5 checks passed          :",
    all_passed,
)

if not all_passed:

    raise RuntimeError(
        "C6-A8-A5 failed. "
        "Do not start the full 315-scenario run."
    )

# Freeze exact bridge objects in memory for the full scenario construction stage.

FD001_FIT_FEATURE_MEAN_628HC6A8A5 = (
    fit_mean.copy()
)

FD001_FIT_FEATURE_STD_628HC6A8A5 = (
    fit_std.copy()
)

FD001_CLEAN_DEV_NORMALIZED_628HC6A8A5 = (
    clean_norm.copy(
        deep=True
    )
)

FD001_CLEAN_DEV_WINDOWS_628HC6A8A5 = (
    X_clean.copy()
)

FD001_CLEAN_DEV_WINDOW_METADATA_628HC6A8A5 = (
    meta_clean.copy(
        deep=True
    )
)

print(
    "\nPASS — FD001 FIT-only normalization + "
    "30-cycle window bridge is locked."
)

print(
    "Next: C6-A8-A6 — construct all 315 corrupted DEV "
    "window tensors only; still no model or MC-dropout execution."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A5 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A6 — FD001 FULL 315-SCENARIO CORRUPTED-WINDOW PATCH BANK
# ==================================================================================

import numpy as np
import pandas as pd
import time

print("=" * 112)
print("STEP 6.28H-C6-A8-A6 — FD001 FULL 315-SCENARIO CORRUPTED-WINDOW PATCH BANK")
print("=" * 112)

# ------------------------------------------------------------------------------
# 1. Require A5 PASS and bind exact locked objects
# ------------------------------------------------------------------------------

a5_audit = C6_A8_A5_AUDIT_628HC6A8A5

if not isinstance(a5_audit, pd.DataFrame):
    raise RuntimeError("C6-A8-A5 audit object is missing or invalid.")

if not bool(a5_audit["passed"].all()):
    raise RuntimeError("C6-A8-A5 must pass before C6-A8-A6.")

manifest = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3
    .copy(deep=True)
    .reset_index(drop=True)
)

dev_base = (
    FD001_DEV_CORRUPTION_BASE_628HC6A8A3
    .copy(deep=True)
)

operator_contract = FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3

features = list(
    FD001_MODEL_FEATURES_628HC6A4
)

fit_mean = (
    FD001_FIT_FEATURE_MEAN_628HC6A8A5
    .copy()
)

fit_std = (
    FD001_FIT_FEATURE_STD_628HC6A8A5
    .copy()
)

clean_windows = np.asarray(
    FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
    dtype=np.float64,
)

clean_metadata = (
    FD001_CLEAN_DEV_WINDOW_METADATA_628HC6A8A5
    .copy(deep=True)
    .reset_index(drop=True)
)

EXPECTED_SCENARIOS = 315
EXPECTED_ROWS = 3852
EXPECTED_WINDOWS = 3272
WINDOW_LENGTH = 30
EXPECTED_FEATURES = 17

if manifest.shape != (EXPECTED_SCENARIOS, 9):
    raise RuntimeError(
        f"Unexpected manifest shape: {manifest.shape}"
    )

if dev_base.shape != (EXPECTED_ROWS, 27):
    raise RuntimeError(
        f"Unexpected canonical DEV-base shape: {dev_base.shape}"
    )

if clean_windows.shape != (
    EXPECTED_WINDOWS,
    WINDOW_LENGTH,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        f"Unexpected clean-window shape: {clean_windows.shape}"
    )

if len(features) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Expected 17 locked features, got {len(features)}."
    )

required_manifest_columns = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
}

if set(manifest.columns) != required_manifest_columns:
    raise RuntimeError(
        "Manifest columns differ from frozen contract. "
        f"Found: {list(manifest.columns)}"
    )

if not np.array_equal(
    dev_base.index.to_numpy(dtype=np.int64),
    np.arange(EXPECTED_ROWS, dtype=np.int64),
):
    raise RuntimeError(
        "Canonical DEV-base index is not RangeIndex(0, 3852)."
    )

feature_index = {
    name: idx
    for idx, name in enumerate(features)
}

retained_sensors = [
    name
    for name in features
    if name.startswith("sensor_")
]

if len(retained_sensors) != 15:
    raise RuntimeError(
        f"Expected 15 retained sensors, got {len(retained_sensors)}."
    )

if set(manifest["sensor"].astype(str)) != set(retained_sensors):
    raise RuntimeError(
        "Manifest sensor set differs from locked retained sensors."
    )

# ------------------------------------------------------------------------------
# 2. Build exact 30-cycle row-position map
# ------------------------------------------------------------------------------

row_windows = []

for _, group in dev_base.groupby(
    "unit_id",
    sort=False,
):

    group_sorted = group.sort_values(
        "cycle",
        kind="mergesort",
    )

    row_ids = group_sorted.index.to_numpy(
        dtype=np.int64
    )

    for start in range(
        len(row_ids) - WINDOW_LENGTH + 1
    ):

        row_windows.append(
            row_ids[
                start:
                start + WINDOW_LENGTH
            ]
        )

window_row_index = np.asarray(
    row_windows,
    dtype=np.int64,
)

if window_row_index.shape != (
    EXPECTED_WINDOWS,
    WINDOW_LENGTH,
):
    raise RuntimeError(
        "Window row-index bridge has wrong shape: "
        f"{window_row_index.shape}"
    )

# ------------------------------------------------------------------------------
# 3. Verify row-map + FIT normalization reproduces all 17 clean channels
# ------------------------------------------------------------------------------

clean_bridge_max_error = 0.0

for feature_name in features:

    j = feature_index[
        feature_name
    ]

    raw_values = dev_base[
        feature_name
    ].to_numpy(
        dtype=np.float64
    )

    normalized_values = (
        raw_values
        -
        float(
            fit_mean.loc[
                feature_name
            ]
        )
    ) / float(
        fit_std.loc[
            feature_name
        ]
    )

    mapped_channel = normalized_values[
        window_row_index
    ]

    channel_error = float(
        np.max(
            np.abs(
                mapped_channel
                -
                clean_windows[
                    :,
                    :,
                    j,
                ]
            )
        )
    )

    clean_bridge_max_error = max(
        clean_bridge_max_error,
        channel_error,
    )

clean_bridge_ok = bool(
    clean_bridge_max_error <= 1e-12
)

if not clean_bridge_ok:

    raise RuntimeError(
        "Lossless row-position bridge failed clean identity. "
        f"max_abs_error={clean_bridge_max_error:.18e}"
    )

# ------------------------------------------------------------------------------
# 4. Exact FD001 scenario application helper
# ------------------------------------------------------------------------------

def apply_fd001_scenario_628hc6a8a6(
    base_raw,
    scenario_row,
):

    family = str(
        scenario_row["family"]
    )

    sensor = str(
        scenario_row["sensor"]
    )

    base_copy = base_raw.copy(
        deep=True
    )

    if family == "bias":

        result = operator_contract["bias"](
            base_copy,
            sensor,
            float(
                scenario_row["alpha"]
            ),
            int(
                scenario_row["direction"]
            ),
        )

    elif family == "drift":

        result = operator_contract["drift"](
            base_copy,
            sensor,
            float(
                scenario_row["alpha"]
            ),
            int(
                scenario_row["direction"]
            ),
        )

    elif family == "noise":

        result = operator_contract["noise"](
            base_copy,
            sensor,
            float(
                scenario_row["alpha"]
            ),
            int(
                scenario_row["seed"]
            ),
        )

    elif family == "dropout":

        result = operator_contract["dropout"](
            base_copy,
            sensor,
            float(
                scenario_row["requested_fraction"]
            ),
            int(
                scenario_row["seed"]
            ),
        )

    elif family == "stuck":

        result = operator_contract["stuck"](
            base_copy,
            sensor,
            float(
                scenario_row["requested_fraction"]
            ),
        )

    else:

        raise ValueError(
            f"Unknown corruption family: {family}"
        )

    corrupted = (
        result[0]
        if isinstance(result, tuple)
        else result
    )

    if not isinstance(
        corrupted,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{family} operator did not return a DataFrame."
        )

    return corrupted

# ------------------------------------------------------------------------------
# 5. Construct all 315 normalized sensor-window patches
# ------------------------------------------------------------------------------

patch_bank = np.empty(
    (
        EXPECTED_SCENARIOS,
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
    ),
    dtype=np.float64,
)

scenario_sensor_feature_index = np.empty(
    EXPECTED_SCENARIOS,
    dtype=np.int16,
)

summary_rows = []

base_columns = list(
    dev_base.columns
)

t0 = time.perf_counter()

for scenario_idx in range(
    EXPECTED_SCENARIOS
):

    row = manifest.iloc[
        scenario_idx
    ]

    family = str(
        row["family"]
    )

    sensor = str(
        row["sensor"]
    )

    severity = str(
        row["severity"]
    )

    corrupted = apply_fd001_scenario_628hc6a8a6(
        dev_base,
        row,
    )

    if corrupted.shape != dev_base.shape:

        raise RuntimeError(
            f"Scenario {scenario_idx}: corrupted shape changed "
            f"from {dev_base.shape} to {corrupted.shape}."
        )

    if list(corrupted.columns) != base_columns:

        raise RuntimeError(
            f"Scenario {scenario_idx}: column order changed."
        )

    sensor_idx = int(
        feature_index[
            sensor
        ]
    )

    scenario_sensor_feature_index[
        scenario_idx
    ] = sensor_idx

    corrupted_sensor_raw = corrupted[
        sensor
    ].to_numpy(
        dtype=np.float64
    )

    clean_sensor_raw = dev_base[
        sensor
    ].to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(
        corrupted_sensor_raw
    ).all():

        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            "non-finite corrupted sensor values."
        )

    normalized_sensor = (
        corrupted_sensor_raw
        -
        float(
            fit_mean.loc[
                sensor
            ]
        )
    ) / float(
        fit_std.loc[
            sensor
        ]
    )

    sensor_patch = normalized_sensor[
        window_row_index
    ]

    if sensor_patch.shape != (
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
    ):

        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            f"patch shape mismatch {sensor_patch.shape}."
        )

    if not np.isfinite(
        sensor_patch
    ).all():

        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            "non-finite normalized patch."
        )

    patch_bank[
        scenario_idx
    ] = sensor_patch

    clean_channel = clean_windows[
        :,
        :,
        sensor_idx,
    ]

    raw_changed_rows = int(
        np.count_nonzero(
            corrupted_sensor_raw
            !=
            clean_sensor_raw
        )
    )

    window_changed_entries = int(
        np.count_nonzero(
            np.abs(
                sensor_patch
                -
                clean_channel
            )
            >
            1e-12
        )
    )

    if window_changed_entries <= 0:

        raise RuntimeError(
            f"Scenario {scenario_idx} produced "
            "no changed window entries."
        )

    summary_rows.append(
        {
            "scenario_index":
                int(
                    scenario_idx
                ),

            "scenario_number":
                int(
                    row["scenario_number"]
                ),

            "scenario_id":
                str(
                    row["scenario_id"]
                ),

            "family":
                family,

            "severity":
                severity,

            "sensor":
                sensor,

            "feature_index":
                sensor_idx,

            "raw_changed_rows":
                raw_changed_rows,

            "window_changed_entries":
                window_changed_entries,

            "patch_min":
                float(
                    np.min(
                        sensor_patch
                    )
                ),

            "patch_median":
                float(
                    np.median(
                        sensor_patch
                    )
                ),

            "patch_max":
                float(
                    np.max(
                        sensor_patch
                    )
                ),
        }
    )

    if (
        scenario_idx == 0
        or
        (scenario_idx + 1) % 25 == 0
        or
        scenario_idx + 1 == EXPECTED_SCENARIOS
    ):

        print(
            f"built {scenario_idx + 1:3d}/{EXPECTED_SCENARIOS} "
            f"| {family:7s} | {severity:6s} | {sensor}"
        )

runtime_sec = float(
    time.perf_counter()
    -
    t0
)

build_summary = pd.DataFrame(
    summary_rows
)

# ------------------------------------------------------------------------------
# 6. Freeze exact in-memory construction objects
# ------------------------------------------------------------------------------

FD001_WINDOW_ROW_INDEX_628HC6A8A6 = (
    window_row_index.copy()
)

FD001_CORRUPTED_SENSOR_WINDOW_PATCHES_628HC6A8A6 = (
    patch_bank
)

FD001_SCENARIO_SENSOR_FEATURE_INDEX_628HC6A8A6 = (
    scenario_sensor_feature_index.copy()
)

FD001_CORRUPTION_BUILD_SUMMARY_628HC6A8A6 = (
    build_summary.copy(
        deep=True
    )
)

FD001_CORRUPTION_MANIFEST_LOCKED_628HC6A8A6 = (
    manifest.copy(
        deep=True
    )
)

# ------------------------------------------------------------------------------
# 7. Exact full-tensor reconstruction helper
# ------------------------------------------------------------------------------

def reconstruct_fd001_corrupted_windows_628hc6a8a6(
    scenario_index,
):

    idx = int(
        scenario_index
    )

    if idx < 0 or idx >= EXPECTED_SCENARIOS:

        raise IndexError(
            f"scenario_index must be in "
            f"[0, {EXPECTED_SCENARIOS - 1}]"
        )

    X = np.asarray(
        FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
        dtype=np.float64,
    ).copy()

    j = int(
        FD001_SCENARIO_SENSOR_FEATURE_INDEX_628HC6A8A6[
            idx
        ]
    )

    X[
        :,
        :,
        j,
    ] = (
        FD001_CORRUPTED_SENSOR_WINDOW_PATCHES_628HC6A8A6[
            idx
        ]
    )

    return X

# ------------------------------------------------------------------------------
# 8. Reconstruction smoke: one scenario per family
# ------------------------------------------------------------------------------

reconstruction_rows = []

for family in [
    "bias",
    "drift",
    "noise",
    "dropout",
    "stuck",
]:

    candidate_idx = int(
        build_summary.index[
            build_summary[
                "family"
            ]
            ==
            family
        ][0]
    )

    X_rebuilt = (
        reconstruct_fd001_corrupted_windows_628hc6a8a6(
            candidate_idx
        )
    )

    j = int(
        scenario_sensor_feature_index[
            candidate_idx
        ]
    )

    other_idx = [
        k
        for k in range(
            EXPECTED_FEATURES
        )
        if k != j
    ]

    reconstruction_rows.append(
        {
            "family":
                family,

            "scenario_index":
                candidate_idx,

            "shape_exact":
                bool(
                    X_rebuilt.shape
                    ==
                    (
                        EXPECTED_WINDOWS,
                        WINDOW_LENGTH,
                        EXPECTED_FEATURES,
                    )
                ),

            "finite":
                bool(
                    np.isfinite(
                        X_rebuilt
                    ).all()
                ),

            "target_patch_exact":
                bool(
                    np.array_equal(
                        X_rebuilt[
                            :,
                            :,
                            j,
                        ],
                        patch_bank[
                            candidate_idx
                        ],
                    )
                ),

            "non_target_exact":
                bool(
                    np.array_equal(
                        X_rebuilt[
                            :,
                            :,
                            other_idx,
                        ],
                        clean_windows[
                            :,
                            :,
                            other_idx,
                        ],
                    )
                ),
        }
    )

reconstruction_audit = pd.DataFrame(
    reconstruction_rows
)

# ------------------------------------------------------------------------------
# 9. Family × severity diagnostics
# ------------------------------------------------------------------------------

family_severity_summary = (
    build_summary
    .groupby(
        [
            "family",
            "severity",
        ],
        sort=True,
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_index",
            "size",
        ),
        median_raw_changed_rows=(
            "raw_changed_rows",
            "median",
        ),
        median_window_changed_entries=(
            "window_changed_entries",
            "median",
        ),
        min_window_changed_entries=(
            "window_changed_entries",
            "min",
        ),
        max_window_changed_entries=(
            "window_changed_entries",
            "max",
        ),
    )
)

print(
    "\n--- LOSSLESS CONSTRUCTION CONTRACT ---"
)

print(
    "Clean row-map bridge max abs error :",
    clean_bridge_max_error,
)

print(
    "Patch bank shape                  :",
    patch_bank.shape,
)

print(
    "Patch bank dtype                  :",
    patch_bank.dtype,
)

print(
    "Patch-bank memory (MiB)           :",
    patch_bank.nbytes / (1024.0 ** 2),
)

print(
    "Construction runtime (sec)        :",
    runtime_sec,
)

print(
    "\n--- FAMILY × SEVERITY BUILD SUMMARY ---"
)

display(
    family_severity_summary
)

print(
    "\n--- FULL-TENSOR RECONSTRUCTION SMOKE ---"
)

display(
    reconstruction_audit
)

# ------------------------------------------------------------------------------
# 10. Final audit
# ------------------------------------------------------------------------------

expected_family_counts = {
    "bias": 90,
    "drift": 90,
    "noise": 45,
    "dropout": 45,
    "stuck": 45,
}

actual_family_counts = (
    build_summary[
        "family"
    ]
    .value_counts()
    .to_dict()
)

audit = pd.DataFrame(
    [
        [
            "previous_step_passed",
            True,
        ],
        [
            "manifest_shape_315x9",
            manifest.shape
            ==
            (
                315,
                9,
            ),
        ],
        [
            "family_counts_exact",
            actual_family_counts
            ==
            expected_family_counts,
        ],
        [
            "window_row_index_shape_3272x30",
            window_row_index.shape
            ==
            (
                3272,
                30,
            ),
        ],
        [
            "clean_row_map_bridge_within_1e12",
            clean_bridge_ok,
        ],
        [
            "patch_bank_shape_315x3272x30",
            patch_bank.shape
            ==
            (
                315,
                3272,
                30,
            ),
        ],
        [
            "patch_bank_finite",
            bool(
                np.isfinite(
                    patch_bank
                ).all()
            ),
        ],
        [
            "sensor_feature_indices_valid",
            bool(
                np.all(
                    (
                        scenario_sensor_feature_index
                        >=
                        0
                    )
                    &
                    (
                        scenario_sensor_feature_index
                        <
                        EXPECTED_FEATURES
                    )
                )
            ),
        ],
        [
            "all_315_scenarios_change_target_windows",
            bool(
                (
                    build_summary[
                        "window_changed_entries"
                    ]
                    >
                    0
                ).all()
            ),
        ],
        [
            "five_family_reconstruction_shapes_exact",
            bool(
                reconstruction_audit[
                    "shape_exact"
                ].all()
            ),
        ],
        [
            "five_family_reconstruction_finite",
            bool(
                reconstruction_audit[
                    "finite"
                ].all()
            ),
        ],
        [
            "five_family_target_patch_exact",
            bool(
                reconstruction_audit[
                    "target_patch_exact"
                ].all()
            ),
        ],
        [
            "five_family_non_target_exact",
            bool(
                reconstruction_audit[
                    "non_target_exact"
                ].all()
            ),
        ],
        [
            "canonical_dev_base_unchanged",
            bool(
                FD001_DEV_CORRUPTION_BASE_628HC6A8A3.equals(
                    dev_base
                )
            ),
        ],
        [
            "clean_window_bank_unchanged",
            bool(
                np.array_equal(
                    FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
                    clean_windows,
                )
            ),
        ],
        [
            "test_features_not_read",
            True,
        ],
        [
            "official_test_labels_not_read",
            True,
        ],
        [
            "test_not_scored",
            True,
        ],
        [
            "no_model_execution",
            True,
        ],
        [
            "no_model_retraining",
            True,
        ],
        [
            "no_mc_dropout_execution",
            True,
        ],
        [
            "no_threshold_modification",
            True,
        ],
        [
            "no_files_written",
            True,
        ],
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A6_AUDIT_628HC6A8A6 = (
    audit
)

all_passed = bool(
    audit[
        "passed"
    ].all()
)

print(
    "\n--- C6-A8-A6 AUDIT ---"
)

display(
    audit
)

print(
    "\n"
    +
    "=" * 112
)

print(
    "STEP 6.28H-C6-A8-A6 FINAL SUMMARY"
)

print(
    "=" * 112
)

print(
    "Scenarios constructed              :",
    len(
        build_summary
    ),
)

print(
    "Lossless patch-bank shape          :",
    patch_bank.shape,
)

print(
    "Clean bridge within 1e-12          :",
    clean_bridge_ok,
)

print(
    "All scenario patches finite        :",
    bool(
        np.isfinite(
            patch_bank
        ).all()
    ),
)

print(
    "All 315 target channels changed    :",
    bool(
        (
            build_summary[
                "window_changed_entries"
            ]
            >
            0
        ).all()
    ),
)

print(
    "Five-family reconstruction exact   :",
    bool(
        reconstruction_audit[
            [
                "shape_exact",
                "finite",
                "target_patch_exact",
                "non_target_exact",
            ]
        ].to_numpy(
            dtype=bool
        ).all()
    ),
)

print(
    "Canonical DEV base unchanged       :",
    bool(
        FD001_DEV_CORRUPTION_BASE_628HC6A8A3.equals(
            dev_base
        )
    ),
)

print(
    "FD001 TEST features read            : False"
)

print(
    "Official FD001 TEST labels read     : False"
)

print(
    "Model executed/retrained            : False"
)

print(
    "MC dropout executed                 : False"
)

print(
    "Threshold modified                  : False"
)

print(
    "Files written                       : False"
)

print(
    "All C6-A8-A6 checks passed          :",
    all_passed,
)

if not all_passed:

    raise RuntimeError(
        "C6-A8-A6 failed. "
        "Do not proceed to corrupted model/MC execution."
    )

print(
    "\nPASS — all 315 corrupted DEV window states are represented "
    "losslessly by the clean tensor + one-channel patch bank."
)

print(
    "Next: C6-A8-A7 — deterministic inference + 50-pass MC dropout "
    "for the 315 frozen corrupted scenarios, one scenario at a time."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A6 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A6-R1 — FD001 315-SCENARIO CORRUPTED-WINDOW PATCH BANK
# ==================================================================================

import numpy as np
import pandas as pd
import time

print("=" * 112)
print("STEP 6.28H-C6-A8-A6-R1 — FD001 315-SCENARIO CORRUPTED-WINDOW PATCH BANK")
print("=" * 112)

# ----------------------------------------------------------------------
# 1. Lock upstream objects
# ----------------------------------------------------------------------

a5_audit = C6_A8_A5_AUDIT_628HC6A8A5

if (
    not isinstance(a5_audit, pd.DataFrame)
    or not bool(a5_audit["passed"].all())
):
    raise RuntimeError(
        "C6-A8-A5 must pass before C6-A8-A6-R1."
    )

manifest = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3
    .copy(deep=True)
    .reset_index(drop=True)
)

dev_base = (
    FD001_DEV_CORRUPTION_BASE_628HC6A8A3
    .copy(deep=True)
)

operator_contract = (
    FD001_CORRUPTION_OPERATOR_CONTRACT_628HC6A8A3
)

features = list(
    FD001_MODEL_FEATURES_628HC6A4
)

fit_mean = (
    FD001_FIT_FEATURE_MEAN_628HC6A8A5
    .copy()
)

fit_std = (
    FD001_FIT_FEATURE_STD_628HC6A8A5
    .copy()
)

clean_windows = np.asarray(
    FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
    dtype=np.float64,
)

clean_metadata = (
    FD001_CLEAN_DEV_WINDOW_METADATA_628HC6A8A5
    .copy(deep=True)
    .reset_index(drop=True)
)

EXPECTED_SCENARIOS = 315
EXPECTED_ROWS = 3852
EXPECTED_WINDOWS = 3272
WINDOW_LENGTH = 30
EXPECTED_FEATURES = 17

if manifest.shape != (315, 9):
    raise RuntimeError(
        f"Unexpected manifest shape: {manifest.shape}"
    )

if dev_base.shape != (3852, 27):
    raise RuntimeError(
        f"Unexpected DEV base shape: {dev_base.shape}"
    )

if clean_windows.shape != (3272, 30, 17):
    raise RuntimeError(
        f"Unexpected clean-window shape: {clean_windows.shape}"
    )

if len(features) != 17:
    raise RuntimeError(
        f"Expected 17 locked features, got {len(features)}."
    )

feature_index = {
    name: i
    for i, name in enumerate(features)
}

retained_sensors = [
    name
    for name in features
    if name.startswith("sensor_")
]

if len(retained_sensors) != 15:
    raise RuntimeError(
        f"Expected 15 retained sensors, got {len(retained_sensors)}."
    )

if set(manifest["sensor"].astype(str)) != set(retained_sensors):
    raise RuntimeError(
        "Manifest sensor set differs from the locked retained-sensor set."
    )

# ----------------------------------------------------------------------
# 2. Rebuild exact 30-cycle row-position bridge
# ----------------------------------------------------------------------

row_windows = []

for _, group in dev_base.groupby(
    "unit_id",
    sort=False,
):

    group_sorted = group.sort_values(
        "cycle",
        kind="mergesort",
    )

    row_ids = group_sorted.index.to_numpy(
        dtype=np.int64
    )

    for start in range(
        len(row_ids) - WINDOW_LENGTH + 1
    ):

        row_windows.append(
            row_ids[
                start:
                start + WINDOW_LENGTH
            ]
        )

window_row_index = np.asarray(
    row_windows,
    dtype=np.int64,
)

if window_row_index.shape != (
    EXPECTED_WINDOWS,
    WINDOW_LENGTH,
):
    raise RuntimeError(
        f"Unexpected row-index bridge shape: "
        f"{window_row_index.shape}"
    )

# ----------------------------------------------------------------------
# 3. Verify exact CLEAN bridge identity
# ----------------------------------------------------------------------

clean_bridge_max_error = 0.0

for feature_name in features:

    j = feature_index[
        feature_name
    ]

    raw = dev_base[
        feature_name
    ].to_numpy(
        dtype=np.float64
    )

    z = (
        raw
        -
        float(
            fit_mean.loc[
                feature_name
            ]
        )
    ) / float(
        fit_std.loc[
            feature_name
        ]
    )

    rebuilt = z[
        window_row_index
    ]

    clean_bridge_max_error = max(
        clean_bridge_max_error,
        float(
            np.max(
                np.abs(
                    rebuilt
                    -
                    clean_windows[
                        :,
                        :,
                        j,
                    ]
                )
            )
        ),
    )

if clean_bridge_max_error > 1e-12:
    raise RuntimeError(
        "Clean bridge identity failed: "
        f"max_abs_error="
        f"{clean_bridge_max_error:.18e}"
    )

# ----------------------------------------------------------------------
# 4. Exact frozen corruption dispatcher
# ----------------------------------------------------------------------

def apply_fd001_scenario_628hc6a8a6r1(
    base_raw,
    row,
):

    family = str(
        row["family"]
    )

    sensor = str(
        row["sensor"]
    )

    work = base_raw.copy(
        deep=True
    )

    if family == "bias":

        result = operator_contract[
            "bias"
        ](
            work,
            sensor,
            float(
                row["alpha"]
            ),
            int(
                row["direction"]
            ),
        )

    elif family == "drift":

        result = operator_contract[
            "drift"
        ](
            work,
            sensor,
            float(
                row["alpha"]
            ),
            int(
                row["direction"]
            ),
        )

    elif family == "noise":

        result = operator_contract[
            "noise"
        ](
            work,
            sensor,
            float(
                row["alpha"]
            ),
            int(
                row["seed"]
            ),
        )

    elif family == "dropout":

        result = operator_contract[
            "dropout"
        ](
            work,
            sensor,
            float(
                row[
                    "requested_fraction"
                ]
            ),
            int(
                row["seed"]
            ),
        )

    elif family == "stuck":

        result = operator_contract[
            "stuck"
        ](
            work,
            sensor,
            float(
                row[
                    "requested_fraction"
                ]
            ),
        )

    else:

        raise ValueError(
            f"Unknown corruption family: {family}"
        )

    corrupted = (
        result[0]
        if isinstance(result, tuple)
        else result
    )

    if not isinstance(
        corrupted,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{family} operator did not return a DataFrame."
        )

    if corrupted.shape != base_raw.shape:
        raise RuntimeError(
            f"{family} operator changed shape: "
            f"{base_raw.shape} -> {corrupted.shape}"
        )

    if list(corrupted.columns) != list(base_raw.columns):
        raise RuntimeError(
            f"{family} operator changed column order."
        )

    return corrupted

# ----------------------------------------------------------------------
# 5. Build all 315 lossless sensor-window patches
# ----------------------------------------------------------------------

patch_bank = np.empty(
    (
        EXPECTED_SCENARIOS,
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
    ),
    dtype=np.float64,
)

scenario_sensor_feature_index = np.empty(
    EXPECTED_SCENARIOS,
    dtype=np.int16,
)

summary_rows = []

t0 = time.perf_counter()

for scenario_idx in range(
    EXPECTED_SCENARIOS
):

    row = manifest.iloc[
        scenario_idx
    ]

    family = str(
        row["family"]
    )

    severity = str(
        row["severity"]
    )

    sensor = str(
        row["sensor"]
    )

    sensor_idx = int(
        feature_index[
            sensor
        ]
    )

    corrupted = (
        apply_fd001_scenario_628hc6a8a6r1(
            dev_base,
            row,
        )
    )

    clean_sensor_raw = dev_base[
        sensor
    ].to_numpy(
        dtype=np.float64
    )

    corrupted_sensor_raw = corrupted[
        sensor
    ].to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(
        corrupted_sensor_raw
    ).all():
        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            "non-finite raw corruption."
        )

    normalized_sensor = (
        corrupted_sensor_raw
        -
        float(
            fit_mean.loc[
                sensor
            ]
        )
    ) / float(
        fit_std.loc[
            sensor
        ]
    )

    sensor_patch = normalized_sensor[
        window_row_index
    ]

    if sensor_patch.shape != (
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
    ):
        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            f"bad patch shape {sensor_patch.shape}."
        )

    if not np.isfinite(
        sensor_patch
    ).all():
        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            "non-finite normalized patch."
        )

    patch_bank[
        scenario_idx
    ] = sensor_patch

    scenario_sensor_feature_index[
        scenario_idx
    ] = sensor_idx

    clean_channel = clean_windows[
        :,
        :,
        sensor_idx,
    ]

    raw_changed_rows = int(
        np.count_nonzero(
            corrupted_sensor_raw
            !=
            clean_sensor_raw
        )
    )

    window_changed_entries = int(
        np.count_nonzero(
            sensor_patch
            !=
            clean_channel
        )
    )

    # --------------------------------------------------------------
    # Correct invariant:
    # A frozen scenario MAY be a realized no-op.
    # But a real raw change must not disappear during windowing.
    # --------------------------------------------------------------

    if (
        raw_changed_rows > 0
        and
        window_changed_entries == 0
    ):
        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            f"raw corruption changed "
            f"{raw_changed_rows} rows but "
            "no window entries changed."
        )

    if (
        raw_changed_rows == 0
        and
        not np.array_equal(
            sensor_patch,
            clean_channel,
        )
    ):
        raise RuntimeError(
            f"Scenario {scenario_idx}: "
            "raw corruption is a no-op "
            "but normalized windows differ from clean."
        )

    summary_rows.append(
        {
            "scenario_index":
                int(
                    scenario_idx
                ),

            "scenario_number":
                int(
                    row[
                        "scenario_number"
                    ]
                ),

            "scenario_id":
                str(
                    row[
                        "scenario_id"
                    ]
                ),

            "family":
                family,

            "severity":
                severity,

            "sensor":
                sensor,

            "feature_index":
                sensor_idx,

            "raw_changed_rows":
                raw_changed_rows,

            "window_changed_entries":
                window_changed_entries,

            "realized_noop":
                bool(
                    raw_changed_rows == 0
                ),
        }
    )

    if (
        scenario_idx == 0
        or
        (scenario_idx + 1) % 25 == 0
        or
        scenario_idx + 1
        ==
        EXPECTED_SCENARIOS
    ):

        print(
            f"built "
            f"{scenario_idx + 1:3d}/"
            f"{EXPECTED_SCENARIOS} "
            f"| {family:7s} "
            f"| {severity:6s} "
            f"| {sensor}"
        )

runtime_sec = float(
    time.perf_counter()
    -
    t0
)

build_summary = pd.DataFrame(
    summary_rows
)

# ----------------------------------------------------------------------
# 6. Freeze exact in-memory construction objects
# ----------------------------------------------------------------------

FD001_WINDOW_ROW_INDEX_628HC6A8A6R1 = (
    window_row_index.copy()
)

FD001_CORRUPTED_SENSOR_WINDOW_PATCHES_628HC6A8A6R1 = (
    patch_bank
)

FD001_SCENARIO_SENSOR_FEATURE_INDEX_628HC6A8A6R1 = (
    scenario_sensor_feature_index.copy()
)

FD001_CORRUPTION_BUILD_SUMMARY_628HC6A8A6R1 = (
    build_summary.copy(
        deep=True
    )
)

FD001_CORRUPTION_MANIFEST_LOCKED_628HC6A8A6R1 = (
    manifest.copy(
        deep=True
    )
)

# ----------------------------------------------------------------------
# 7. Full-tensor reconstruction helper
# ----------------------------------------------------------------------

def reconstruct_fd001_corrupted_windows_628hc6a8a6r1(
    scenario_index,
):

    idx = int(
        scenario_index
    )

    if (
        idx < 0
        or
        idx >= EXPECTED_SCENARIOS
    ):
        raise IndexError(
            "scenario_index must be in "
            f"[0, {EXPECTED_SCENARIOS - 1}]."
        )

    X = clean_windows.copy()

    j = int(
        scenario_sensor_feature_index[
            idx
        ]
    )

    X[
        :,
        :,
        j,
    ] = patch_bank[
        idx
    ]

    return X

# ----------------------------------------------------------------------
# 8. One reconstruction test per family
# ----------------------------------------------------------------------

reconstruction_rows = []

for family in [
    "bias",
    "drift",
    "noise",
    "dropout",
    "stuck",
]:

    idx = int(
        build_summary.index[
            build_summary[
                "family"
            ]
            ==
            family
        ][0]
    )

    X = (
        reconstruct_fd001_corrupted_windows_628hc6a8a6r1(
            idx
        )
    )

    j = int(
        scenario_sensor_feature_index[
            idx
        ]
    )

    other = np.asarray(
        [
            k
            for k in range(
                EXPECTED_FEATURES
            )
            if k != j
        ],
        dtype=np.int64,
    )

    reconstruction_rows.append(
        {
            "family":
                family,

            "scenario_index":
                idx,

            "shape_exact":
                bool(
                    X.shape
                    ==
                    (
                        3272,
                        30,
                        17,
                    )
                ),

            "finite":
                bool(
                    np.isfinite(
                        X
                    ).all()
                ),

            "target_patch_exact":
                bool(
                    np.array_equal(
                        X[
                            :,
                            :,
                            j,
                        ],
                        patch_bank[
                            idx
                        ],
                    )
                ),

            "non_target_exact":
                bool(
                    np.array_equal(
                        X[
                            :,
                            :,
                            other,
                        ],
                        clean_windows[
                            :,
                            :,
                            other,
                        ],
                    )
                ),
        }
    )

reconstruction_audit = pd.DataFrame(
    reconstruction_rows
)

FD001_CORRUPTION_RECONSTRUCTION_AUDIT_628HC6A8A6R1 = (
    reconstruction_audit.copy(
        deep=True
    )
)

# ----------------------------------------------------------------------
# 9. Explicitly report realized no-op scenarios
# ----------------------------------------------------------------------

noop_scenarios = (
    build_summary.loc[
        build_summary[
            "realized_noop"
        ],
        [
            "scenario_index",
            "scenario_number",
            "scenario_id",
            "family",
            "severity",
            "sensor",
            "raw_changed_rows",
            "window_changed_entries",
        ],
    ]
    .reset_index(
        drop=True
    )
)

family_severity_summary = (
    build_summary
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_index",
            "size",
        ),
        realized_noops=(
            "realized_noop",
            "sum",
        ),
        median_raw_changed_rows=(
            "raw_changed_rows",
            "median",
        ),
        median_window_changed_entries=(
            "window_changed_entries",
            "median",
        ),
    )
)

print(
    "\n--- FAMILY × SEVERITY BUILD SUMMARY ---"
)

display(
    family_severity_summary
)

print(
    "\n--- REALIZED NO-OP SCENARIOS ---"
)

if len(noop_scenarios) == 0:

    print(
        "None"
    )

else:

    display(
        noop_scenarios
    )

print(
    "\n--- FIVE-FAMILY RECONSTRUCTION AUDIT ---"
)

display(
    reconstruction_audit
)

# ----------------------------------------------------------------------
# 10. Final audit
# ----------------------------------------------------------------------

expected_family_counts = {
    "bias": 90,
    "drift": 90,
    "noise": 45,
    "dropout": 45,
    "stuck": 45,
}

family_counts_exact = (
    build_summary[
        "family"
    ]
    .value_counts()
    .to_dict()
    ==
    expected_family_counts
)

raw_change_propagation_exact = bool(
    (
        (
            build_summary[
                "raw_changed_rows"
            ]
            ==
            0
        )
        ==
        (
            build_summary[
                "window_changed_entries"
            ]
            ==
            0
        )
    ).all()
)

reconstruction_exact = bool(
    reconstruction_audit[
        [
            "shape_exact",
            "finite",
            "target_patch_exact",
            "non_target_exact",
        ]
    ]
    .to_numpy(
        dtype=bool
    )
    .all()
)

audit = pd.DataFrame(
    [
        [
            "previous_step_passed",
            True,
        ],
        [
            "manifest_shape_315x9",
            manifest.shape
            ==
            (
                315,
                9,
            ),
        ],
        [
            "family_counts_exact",
            family_counts_exact,
        ],
        [
            "window_row_index_shape_3272x30",
            window_row_index.shape
            ==
            (
                3272,
                30,
            ),
        ],
        [
            "clean_bridge_identity_within_1e12",
            clean_bridge_max_error
            <=
            1e-12,
        ],
        [
            "patch_bank_shape_315x3272x30",
            patch_bank.shape
            ==
            (
                315,
                3272,
                30,
            ),
        ],
        [
            "patch_bank_finite",
            bool(
                np.isfinite(
                    patch_bank
                ).all()
            ),
        ],
        [
            "all_315_scenarios_represented",
            len(
                build_summary
            )
            ==
            315,
        ],
        [
            "raw_change_to_window_change_consistent",
            raw_change_propagation_exact,
        ],
        [
            "realized_noops_retained_not_relabelled",
            True,
        ],
        [
            "five_family_full_tensor_reconstruction_exact",
            reconstruction_exact,
        ],
        [
            "canonical_dev_base_unchanged",
            bool(
                FD001_DEV_CORRUPTION_BASE_628HC6A8A3.equals(
                    dev_base
                )
            ),
        ],
        [
            "clean_window_bank_unchanged",
            bool(
                np.array_equal(
                    FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
                    clean_windows,
                )
            ),
        ],
        [
            "test_features_not_read",
            True,
        ],
        [
            "official_test_labels_not_read",
            True,
        ],
        [
            "test_not_scored",
            True,
        ],
        [
            "no_model_execution",
            True,
        ],
        [
            "no_model_retraining",
            True,
        ],
        [
            "no_mc_dropout_execution",
            True,
        ],
        [
            "no_threshold_modification",
            True,
        ],
        [
            "no_files_written",
            True,
        ],
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A6_R1_AUDIT_628HC6A8A6R1 = (
    audit
)

all_passed = bool(
    audit[
        "passed"
    ].all()
)

print(
    "\n--- C6-A8-A6-R1 AUDIT ---"
)

display(
    audit
)

print(
    "\n"
    +
    "=" * 112
)

print(
    "STEP 6.28H-C6-A8-A6-R1 FINAL SUMMARY"
)

print(
    "=" * 112
)

print(
    "Scenarios represented              :",
    len(
        build_summary
    ),
)

print(
    "Patch-bank shape                   :",
    patch_bank.shape,
)

print(
    "Patch-bank memory (MiB)            :",
    patch_bank.nbytes
    /
    (1024.0 ** 2),
)

print(
    "Clean bridge max abs error         :",
    clean_bridge_max_error,
)

print(
    "Realized no-op scenarios           :",
    len(
        noop_scenarios
    ),
)

print(
    "Raw/window change consistency      :",
    raw_change_propagation_exact,
)

print(
    "Five-family reconstruction exact   :",
    reconstruction_exact,
)

print(
    "Construction runtime (sec)         :",
    runtime_sec,
)

print(
    "FD001 TEST features read           : False"
)

print(
    "Official FD001 TEST labels read    : False"
)

print(
    "Model executed/retrained           : False"
)

print(
    "MC dropout executed                : False"
)

print(
    "Threshold modified                 : False"
)

print(
    "Files written                      : False"
)

print(
    "All C6-A8-A6-R1 checks passed      :",
    all_passed,
)

if not all_passed:

    raise RuntimeError(
        "C6-A8-A6-R1 failed. "
        "Do not proceed to corrupted model/MC execution."
    )

print(
    "\nPASS — all 315 frozen scenarios are represented exactly. "
    "Any realized no-op scenarios are preserved as part of the frozen protocol."
)

print(
    "Next: C6-A8-A7 — deterministic inference + 50-pass MC dropout "
    "for the 315 frozen corrupted scenarios, using the existing frozen threshold."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A6-R1 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A7 — FD001 FULL 315-SCENARIO DETERMINISTIC + 50-PASS MC DROPOUT
# ==================================================================================

import hashlib
import time
import numpy as np
import pandas as pd
import torch

print("=" * 116)
print("STEP 6.28H-C6-A8-A7 — FD001 FULL 315-SCENARIO DETERMINISTIC + 50-PASS MC DROPOUT")
print("=" * 116)

# ----------------------------- frozen upstream contract -----------------------------

audit_prev = C6_A8_A6_R1_AUDIT_628HC6A8A6R1

if (
    not isinstance(audit_prev, pd.DataFrame)
    or "passed" not in audit_prev.columns
    or not bool(audit_prev["passed"].all())
):
    raise RuntimeError("C6-A8-A6-R1 must pass before C6-A8-A7.")

model = MODEL_FD001_628HC6A6A1

manifest = (
    FD001_CORRUPTION_MANIFEST_LOCKED_628HC6A8A6R1
    .reset_index(drop=True)
)

build_summary = (
    FD001_CORRUPTION_BUILD_SUMMARY_628HC6A8A6R1
    .reset_index(drop=True)
)

if manifest.shape != (315, 9):
    raise RuntimeError(
        f"Unexpected manifest shape: {manifest.shape}"
    )

if len(build_summary) != 315:
    raise RuntimeError(
        f"Unexpected build-summary length: {len(build_summary)}"
    )

if not callable(
    reconstruct_fd001_corrupted_windows_628hc6a8a6r1
):
    raise RuntimeError(
        "Corrupted-window reconstruction helper is unavailable."
    )

if not callable(
    mc_dropout_predict_628hc0
):
    raise RuntimeError(
        "Frozen MC-dropout helper is unavailable."
    )

N_SCENARIOS = 315
N_WINDOWS = 3272
WINDOW = 30
N_FEATURES = 17

MC_PASSES = 50
MC_BATCH = 512
MC_SEED = 42
DET_BATCH = 512


# ----------------------------- model fingerprint ------------------------------------

def model_sha256_628hc6a8a7(model_obj):

    h = hashlib.sha256()

    for key, tensor in model_obj.state_dict().items():

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            str(key).encode("utf-8")
        )

        h.update(
            str(arr.dtype).encode("utf-8")
        )

        h.update(
            np.asarray(
                arr.shape,
                dtype=np.int64,
            ).tobytes()
        )

        h.update(
            arr.tobytes(order="C")
        )

    return h.hexdigest()


sha_before = model_sha256_628hc6a8a7(
    model
)

mode_before = bool(
    model.training
)

device = next(
    model.parameters()
).device

print(
    "Model device :",
    device,
)

print(
    "Model SHA-256:",
    sha_before,
)


# ----------------------------- deterministic inference ------------------------------

def predict_fd001_det_628hc6a8a7(X):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    if X.shape != (
        N_WINDOWS,
        WINDOW,
        N_FEATURES,
    ):
        raise ValueError(
            f"Unexpected deterministic input shape: {X.shape}"
        )

    incoming_mode = bool(
        model.training
    )

    model.eval()

    out = []

    try:

        with torch.no_grad():

            for start in range(
                0,
                N_WINDOWS,
                DET_BATCH,
            ):

                stop = min(
                    start + DET_BATCH,
                    N_WINDOWS,
                )

                xb = torch.as_tensor(
                    X[start:stop],
                    dtype=torch.float32,
                    device=device,
                )

                yb = model(
                    xb
                )

                if isinstance(
                    yb,
                    (tuple, list),
                ):

                    if len(yb) != 1:
                        raise RuntimeError(
                            "Unexpected deterministic model "
                            "tuple/list output."
                        )

                    yb = yb[0]

                pred = (
                    yb
                    .detach()
                    .cpu()
                    .numpy()
                    .reshape(-1)
                )

                if pred.size != (
                    stop - start
                ):
                    raise RuntimeError(
                        f"Bad deterministic batch size: "
                        f"{pred.size}, expected {stop-start}."
                    )

                out.append(
                    pred.astype(
                        np.float64,
                        copy=False,
                    )
                )

    finally:

        model.train(
            incoming_mode
        )

    result = np.concatenate(
        out
    )

    if result.shape != (
        N_WINDOWS,
    ):
        raise RuntimeError(
            f"Unexpected deterministic result shape: "
            f"{result.shape}"
        )

    if not np.isfinite(
        result
    ).all():
        raise RuntimeError(
            "Deterministic predictions contain "
            "non-finite values."
        )

    return result


# ----------------------------- allocate outputs -------------------------------------

det_matrix = np.empty(
    (
        N_SCENARIOS,
        N_WINDOWS,
    ),
    dtype=np.float64,
)

mc_mean_matrix = np.empty(
    (
        N_SCENARIOS,
        N_WINDOWS,
    ),
    dtype=np.float64,
)

mc_std_matrix = np.empty(
    (
        N_SCENARIOS,
        N_WINDOWS,
    ),
    dtype=np.float64,
)

runtime_rows = []

t_all = time.perf_counter()


# ----------------------------- full 315-scenario execution ---------------------------

for i in range(
    N_SCENARIOS
):

    row = manifest.iloc[
        i
    ]

    t0 = time.perf_counter()

    X = np.asarray(
        reconstruct_fd001_corrupted_windows_628hc6a8a6r1(
            i
        ),
        dtype=np.float64,
    )

    if X.shape != (
        N_WINDOWS,
        WINDOW,
        N_FEATURES,
    ):
        raise RuntimeError(
            f"Scenario {i}: bad tensor shape {X.shape}."
        )

    if not np.isfinite(
        X
    ).all():
        raise RuntimeError(
            f"Scenario {i}: non-finite model input."
        )

    # deterministic
    det = predict_fd001_det_628hc6a8a7(
        X
    )

    # frozen 50-pass MC dropout
    mc_out = mc_dropout_predict_628hc0(
        X,
        model,
        passes=MC_PASSES,
        batch_size=MC_BATCH,
        seed=MC_SEED,
    )

    if (
        not isinstance(
            mc_out,
            (tuple, list),
        )
        or len(mc_out) != 2
    ):
        raise RuntimeError(
            f"Scenario {i}: MC helper did not "
            "return (mean, std)."
        )

    mc_mean = np.asarray(
        mc_out[0],
        dtype=np.float64,
    ).reshape(-1)

    mc_std = np.asarray(
        mc_out[1],
        dtype=np.float64,
    ).reshape(-1)

    if (
        mc_mean.shape
        !=
        (N_WINDOWS,)
        or
        not np.isfinite(
            mc_mean
        ).all()
    ):
        raise RuntimeError(
            f"Scenario {i}: invalid MC-mean vector."
        )

    if (
        mc_std.shape
        !=
        (N_WINDOWS,)
        or
        not np.isfinite(
            mc_std
        ).all()
    ):
        raise RuntimeError(
            f"Scenario {i}: invalid MC-std vector."
        )

    if np.any(
        mc_std < 0.0
    ):
        raise RuntimeError(
            f"Scenario {i}: negative MC standard deviation."
        )

    det_matrix[
        i
    ] = det

    mc_mean_matrix[
        i
    ] = mc_mean

    mc_std_matrix[
        i
    ] = mc_std

    # Restore incoming model mode after stochastic helper.
    model.train(
        mode_before
    )

    elapsed = float(
        time.perf_counter()
        -
        t0
    )

    runtime_rows.append(
        {
            "scenario_index":
                i,

            "scenario_number":
                int(
                    row[
                        "scenario_number"
                    ]
                ),

            "scenario_id":
                str(
                    row[
                        "scenario_id"
                    ]
                ),

            "family":
                str(
                    row[
                        "family"
                    ]
                ),

            "severity":
                str(
                    row[
                        "severity"
                    ]
                ),

            "sensor":
                str(
                    row[
                        "sensor"
                    ]
                ),

            "realized_noop":
                bool(
                    build_summary.loc[
                        i,
                        "realized_noop",
                    ]
                ),

            "mc_std_median":
                float(
                    np.median(
                        mc_std
                    )
                ),

            "runtime_sec":
                elapsed,
        }
    )

    if (
        i == 0
        or
        (i + 1) % 10 == 0
        or
        (i + 1) == N_SCENARIOS
    ):

        print(
            f"done {i+1:3d}/{N_SCENARIOS} | "
            f"{str(row['family']):7s} | "
            f"{str(row['severity']):6s} | "
            f"{str(row['sensor']):9s} | "
            f"MC-STD med {np.median(mc_std):.6f} | "
            f"{elapsed:.2f}s"
        )


total_runtime = float(
    time.perf_counter()
    -
    t_all
)


# ----------------------------- no-op identity ---------------------------------------

noop_indices = (
    build_summary.loc[
        build_summary[
            "realized_noop"
        ],
        "scenario_index",
    ]
    .astype(int)
    .tolist()
)

clean_windows = np.asarray(
    FD001_CLEAN_DEV_WINDOWS_628HC6A8A5,
    dtype=np.float64,
)

noop_input_identity = all(
    np.array_equal(
        np.asarray(
            reconstruct_fd001_corrupted_windows_628hc6a8a6r1(
                i
            ),
            dtype=np.float64,
        ),
        clean_windows,
    )
    for i in noop_indices
)


# ----------------------------- freeze in-memory outputs ------------------------------

runtime_df = pd.DataFrame(
    runtime_rows
)

FD001_CORRUPTED_DETERMINISTIC_MATRIX_628HC6A8A7 = (
    det_matrix
)

FD001_CORRUPTED_MC_MEAN_MATRIX_628HC6A8A7 = (
    mc_mean_matrix
)

FD001_CORRUPTED_MC_STD_MATRIX_628HC6A8A7 = (
    mc_std_matrix
)

FD001_CORRUPTED_MC_RUNTIME_628HC6A8A7 = (
    runtime_df.copy(
        deep=True
    )
)


# ----------------------------- final audit ------------------------------------------

model.train(
    mode_before
)

sha_after = model_sha256_628hc6a8a7(
    model
)

audit = pd.DataFrame(
    [
        (
            "previous_step_passed",
            True,
        ),
        (
            "det_shape_315x3272",
            det_matrix.shape
            ==
            (
                315,
                3272,
            ),
        ),
        (
            "mc_mean_shape_315x3272",
            mc_mean_matrix.shape
            ==
            (
                315,
                3272,
            ),
        ),
        (
            "mc_std_shape_315x3272",
            mc_std_matrix.shape
            ==
            (
                315,
                3272,
            ),
        ),
        (
            "det_finite",
            bool(
                np.isfinite(
                    det_matrix
                ).all()
            ),
        ),
        (
            "mc_mean_finite",
            bool(
                np.isfinite(
                    mc_mean_matrix
                ).all()
            ),
        ),
        (
            "mc_std_finite",
            bool(
                np.isfinite(
                    mc_std_matrix
                ).all()
            ),
        ),
        (
            "mc_std_nonnegative",
            bool(
                (
                    mc_std_matrix
                    >=
                    0.0
                ).all()
            ),
        ),
        (
            "all_315_scenarios_executed",
            len(
                runtime_df
            )
            ==
            315,
        ),
        (
            "mc_passes_50",
            MC_PASSES
            ==
            50,
        ),
        (
            "mc_batch_size_512",
            MC_BATCH
            ==
            512,
        ),
        (
            "mc_seed_42",
            MC_SEED
            ==
            42,
        ),
        (
            "common_seed_across_scenarios",
            True,
        ),
        (
            "realized_noop_inputs_equal_clean",
            bool(
                noop_input_identity
            ),
        ),
        (
            "model_state_sha256_unchanged",
            sha_before
            ==
            sha_after,
        ),
        (
            "model_training_mode_restored",
            bool(
                model.training
            )
            ==
            mode_before,
        ),
        (
            "test_features_not_read",
            True,
        ),
        (
            "official_test_labels_not_read",
            True,
        ),
        (
            "test_not_scored",
            True,
        ),
        (
            "no_model_retraining",
            True,
        ),
        (
            "no_threshold_modification",
            True,
        ),
        (
            "no_files_written",
            True,
        ),
    ],
    columns=[
        "check",
        "passed",
    ],
)

C6_A8_A7_AUDIT_628HC6A8A7 = (
    audit
)

all_passed = bool(
    audit[
        "passed"
    ].all()
)

print(
    "\n--- C6-A8-A7 AUDIT ---"
)

display(
    audit
)

print(
    "\n"
    +
    "=" * 116
)

print(
    "STEP 6.28H-C6-A8-A7 FINAL SUMMARY"
)

print(
    "=" * 116
)

print(
    "Scenarios executed                 :",
    len(
        runtime_df
    ),
)

print(
    "Deterministic matrix shape         :",
    det_matrix.shape,
)

print(
    "MC mean matrix shape               :",
    mc_mean_matrix.shape,
)

print(
    "MC std matrix shape                :",
    mc_std_matrix.shape,
)

print(
    "MC passes / batch / seed           :",
    MC_PASSES,
    "/",
    MC_BATCH,
    "/",
    MC_SEED,
)

print(
    "Median scenario MC-STD             :",
    float(
        np.median(
            np.median(
                mc_std_matrix,
                axis=1,
            )
        )
    ),
)

print(
    "Maximum observed MC-STD            :",
    float(
        np.max(
            mc_std_matrix
        )
    ),
)

print(
    "Realized no-op input identity      :",
    noop_input_identity,
)

print(
    "Model-state SHA-256 unchanged      :",
    sha_before
    ==
    sha_after,
)

print(
    "Model training mode restored       :",
    bool(
        model.training
    )
    ==
    mode_before,
)

print(
    "Total runtime (sec)                :",
    total_runtime,
)

print(
    "FD001 TEST features read           : False"
)

print(
    "Official FD001 TEST labels read    : False"
)

print(
    "TEST scored                        : False"
)

print(
    "Model retrained                    : False"
)

print(
    "Threshold modified                 : False"
)

print(
    "Files written                      : False"
)

print(
    "All C6-A8-A7 checks passed         :",
    all_passed,
)

if not all_passed:

    raise RuntimeError(
        "C6-A8-A7 failed. "
        "Do not proceed to corrupted-risk calibration."
    )

print(
    "\nPASS — deterministic and 50-pass MC-dropout outputs "
    "are complete for all 315 frozen FD001 DEV corruption scenarios."
)

print(
    "Next: C6-A8-A8 — apply the already-frozen FD001 "
    "reliability calibration and threshold, then evaluate "
    "selective DEV behavior."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A7 ==="
)

In [ ]:
# ==================================================================================
# STEP 6.28H-C6-A8-A7-R0 — MC HELPER RETURN-CONTRACT INSPECTION (NO MODEL EXECUTION)
# ==================================================================================

import inspect
import numpy as np

print("=" * 116)
print("STEP 6.28H-C6-A8-A7-R0 — MC HELPER RETURN-CONTRACT INSPECTION")
print("=" * 116)

print("\n--- HELPER SOURCE ---")
print("name      :", mc_dropout_predict_628hc0.__name__)
print("signature :", inspect.signature(mc_dropout_predict_628hc0))
print(inspect.getsource(mc_dropout_predict_628hc0))

print("\n--- EXISTING scenario-0 mc_out OBJECT ---")
print("type      :", type(mc_out).__name__)

if isinstance(mc_out, dict):
    print("keys      :", list(mc_out.keys()))

    for k, v in mc_out.items():
        a = np.asarray(v)

        print(
            f"{k!r}: "
            f"type={type(v).__name__}, "
            f"shape={a.shape}, "
            f"dtype={a.dtype}"
        )

elif isinstance(mc_out, (tuple, list)):
    print("length    :", len(mc_out))

    for j, v in enumerate(mc_out):
        a = np.asarray(v)

        print(
            f"[{j}]: "
            f"type={type(v).__name__}, "
            f"shape={a.shape}, "
            f"dtype={a.dtype}"
        )

else:
    a = np.asarray(mc_out)

    print("array shape:", a.shape)
    print("array dtype:", a.dtype)

print("\n--- SAFETY ---")
print("Additional model execution : False")
print("Additional MC dropout       : False")
print("Model retraining            : False")
print("TEST features read          : False")
print("Official TEST labels read   : False")
print("Threshold modified          : False")
print("Files written               : False")

print("\nSend this output back. Do not rerun C6-A8-A7 yet.")

In [ ]:
# ==================================================================================================
# STEP 6.28H-C6-A8-A7-R1 — FD001 FULL 315-SCENARIO DETERMINISTIC + 50-PASS MC DROPOUT
# Corrected MC-helper contract: (mc_mean, mc_std, runtime)
# ==================================================================================================

import time
import hashlib
import numpy as np
import pandas as pd
import torch

print("=" * 118)
print("STEP 6.28H-C6-A8-A7-R1 — FD001 FULL 315-SCENARIO DETERMINISTIC + 50-PASS MC DROPOUT")
print("=" * 118)

# --------------------------------------------------------------------------------------------------
# 0. Frozen execution constants
# --------------------------------------------------------------------------------------------------
EXPECTED_SCENARIOS = 315
EXPECTED_WINDOWS = 3272
WINDOW_LENGTH = 30
FEATURE_COUNT = 17

MC_PASSES = 50
MC_BATCH_SIZE = 512
MC_SEED = 42
DET_BATCH_SIZE = 512

# --------------------------------------------------------------------------------------------------
# 1. Resolve and validate the exact locked objects
# --------------------------------------------------------------------------------------------------
required_names = [
    "MODEL_FD001_628HC6A6A1",
    "FD001_X_DEV_WINDOWS_628HC6A5R1",
    "FD001_MODEL_FEATURES_628HC6A4",
    "FD001_CORRUPTION_MANIFEST_628HC6A8A3",
    "mc_dropout_predict_628hc0",
]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required locked object(s): "
        + ", ".join(missing)
    )

model = MODEL_FD001_628HC6A6A1

clean_x = np.asarray(
    FD001_X_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float64,
)

model_features = list(
    FD001_MODEL_FEATURES_628HC6A4
)

manifest = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3
    .reset_index(drop=True)
    .copy()
)

if clean_x.shape != (
    EXPECTED_WINDOWS,
    WINDOW_LENGTH,
    FEATURE_COUNT,
):
    raise RuntimeError(
        f"Locked CLEAN DEV windows have shape "
        f"{clean_x.shape}; expected "
        f"{(EXPECTED_WINDOWS, WINDOW_LENGTH, FEATURE_COUNT)}."
    )

if not np.isfinite(clean_x).all():
    raise RuntimeError(
        "Locked CLEAN DEV windows contain non-finite values."
    )

if len(model_features) != FEATURE_COUNT:
    raise RuntimeError(
        f"Locked model-feature count is "
        f"{len(model_features)}; expected {FEATURE_COUNT}."
    )

if manifest.shape != (
    EXPECTED_SCENARIOS,
    9,
):
    raise RuntimeError(
        f"FD001 corruption manifest has shape "
        f"{manifest.shape}; expected "
        f"{(EXPECTED_SCENARIOS, 9)}."
    )

required_manifest_columns = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
}

if set(manifest.columns) != required_manifest_columns:
    raise RuntimeError(
        "FD001 corruption manifest columns differ from "
        "the frozen 9-column contract.\n"
        f"Observed: {list(manifest.columns)}"
    )

manifest_sensors = set(
    manifest["sensor"].astype(str)
)

missing_manifest_sensors = sorted(
    manifest_sensors.difference(
        model_features
    )
)

if missing_manifest_sensors:
    raise RuntimeError(
        "Manifest sensor(s) are absent from the locked "
        "17-feature model contract: "
        + ", ".join(missing_manifest_sensors)
    )

# --------------------------------------------------------------------------------------------------
# 2. Resolve the exact A6-R1 patch bank
# --------------------------------------------------------------------------------------------------
preferred_patch_names = [
    "FD001_CORRUPTED_WINDOW_PATCH_BANK_628HC6A8A6R1",
    "FD001_WINDOW_PATCH_BANK_628HC6A8A6R1",
    "FD001_CORRUPTED_PATCH_BANK_628HC6A8A6R1",
    "FD001_PATCH_BANK_628HC6A8A6R1",
]

patch_name = None
patch_bank = None

for candidate_name in preferred_patch_names:

    candidate = globals().get(
        candidate_name
    )

    if (
        isinstance(candidate, np.ndarray)
        and candidate.shape
        ==
        (
            EXPECTED_SCENARIOS,
            EXPECTED_WINDOWS,
            WINDOW_LENGTH,
        )
    ):
        patch_name = candidate_name
        patch_bank = candidate
        break

# Safe fallback if the exact variable spelling differs.
if patch_bank is None:

    snapshot = list(
        globals().items()
    )

    stage_candidates = []

    for name, obj in snapshot:

        if (
            isinstance(obj, np.ndarray)
            and obj.shape
            ==
            (
                EXPECTED_SCENARIOS,
                EXPECTED_WINDOWS,
                WINDOW_LENGTH,
            )
            and "628HC6A8A6" in name.upper()
        ):
            stage_candidates.append(
                (name, obj)
            )

    if not stage_candidates:

        for name, obj in snapshot:

            if (
                isinstance(obj, np.ndarray)
                and obj.shape
                ==
                (
                    EXPECTED_SCENARIOS,
                    EXPECTED_WINDOWS,
                    WINDOW_LENGTH,
                )
                and "PATCH" in name.upper()
            ):
                stage_candidates.append(
                    (name, obj)
                )

    if not stage_candidates:
        raise RuntimeError(
            "Could not resolve the locked A6-R1 patch bank "
            "with shape "
            f"{(EXPECTED_SCENARIOS, EXPECTED_WINDOWS, WINDOW_LENGTH)}."
        )

    if len(stage_candidates) == 1:

        patch_name, patch_bank = (
            stage_candidates[0]
        )

    else:

        candidate_hashes = {}

        for name, obj in stage_candidates:

            contiguous = np.ascontiguousarray(
                obj
            )

            candidate_hashes[name] = (
                hashlib.sha256(
                    contiguous.view(
                        np.uint8
                    )
                ).hexdigest()
            )

        unique_hashes = set(
            candidate_hashes.values()
        )

        if len(unique_hashes) != 1:
            raise RuntimeError(
                "Multiple non-identical A6 patch-bank "
                "candidates were found. Refusing to choose silently.\n"
                +
                "\n".join(
                    f"{name}: {digest}"
                    for name, digest
                    in candidate_hashes.items()
                )
            )

        patch_name, patch_bank = sorted(
            stage_candidates,
            key=lambda item: (
                "PATCH_BANK"
                not in item[0].upper(),
                "FD001"
                not in item[0].upper(),
                item[0],
            ),
        )[0]

patch_bank = np.asarray(
    patch_bank,
    dtype=np.float64,
)

if patch_bank.shape != (
    EXPECTED_SCENARIOS,
    EXPECTED_WINDOWS,
    WINDOW_LENGTH,
):
    raise RuntimeError(
        f"Resolved patch bank has wrong shape: "
        f"{patch_bank.shape}"
    )

if not np.isfinite(
    patch_bank
).all():
    raise RuntimeError(
        "Resolved patch bank contains non-finite values."
    )

# --------------------------------------------------------------------------------------------------
# 3. Model/device lock + state hash
# --------------------------------------------------------------------------------------------------
if not isinstance(
    model,
    torch.nn.Module,
):
    raise RuntimeError(
        f"MODEL_FD001_628HC6A6A1 is "
        f"{type(model).__name__}, "
        "not torch.nn.Module."
    )

parameter_count = int(
    sum(
        p.numel()
        for p in model.parameters()
    )
)

if parameter_count != 18049:
    raise RuntimeError(
        f"FD001 model parameter count is "
        f"{parameter_count}; expected 18049."
    )

try:
    model_device = next(
        model.parameters()
    ).device

except StopIteration as exc:
    raise RuntimeError(
        "FD001 model has no parameters."
    ) from exc


def model_state_sha256_628hc6a8a7r1(
    model_obj
):
    h = hashlib.sha256()

    state = model_obj.state_dict()

    for key in sorted(
        state.keys()
    ):

        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        h.update(
            key.encode("utf-8")
        )

        h.update(
            str(
                tuple(
                    tensor.shape
                )
            ).encode("utf-8")
        )

        h.update(
            str(
                tensor.dtype
            ).encode("utf-8")
        )

        h.update(
            tensor.numpy()
            .tobytes(
                order="C"
            )
        )

    return h.hexdigest()


model_sha_before = (
    model_state_sha256_628hc6a8a7r1(
        model
    )
)

original_training_state = bool(
    model.training
)

print(
    f"Patch bank   : {patch_name}"
)

print(
    f"Patch shape  : {patch_bank.shape}"
)

print(
    f"Model device : {model_device}"
)

print(
    f"Parameters   : {parameter_count}"
)

print(
    f"Model SHA-256: {model_sha_before}"
)

# --------------------------------------------------------------------------------------------------
# 4. Deterministic inference helper
# --------------------------------------------------------------------------------------------------
def deterministic_predict_628hc6a8a7r1(
    X_values
):

    X_values = np.asarray(
        X_values,
        dtype=np.float32,
    )

    if X_values.shape != (
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
        FEATURE_COUNT,
    ):
        raise RuntimeError(
            f"Deterministic input shape "
            f"{X_values.shape}; expected "
            f"{(EXPECTED_WINDOWS, WINDOW_LENGTH, FEATURE_COUNT)}."
        )

    out = np.empty(
        EXPECTED_WINDOWS,
        dtype=np.float64,
    )

    model.eval()

    with torch.no_grad():

        for start in range(
            0,
            EXPECTED_WINDOWS,
            DET_BATCH_SIZE,
        ):

            end = min(
                start + DET_BATCH_SIZE,
                EXPECTED_WINDOWS,
            )

            xb = torch.as_tensor(
                X_values[
                    start:end
                ],
                dtype=torch.float32,
                device=model_device,
            )

            pred = (
                model(
                    xb
                )
                .reshape(-1)
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            if pred.shape != (
                end - start,
            ):
                raise RuntimeError(
                    f"Deterministic batch output shape "
                    f"{pred.shape}; expected "
                    f"{(end - start,)}."
                )

            out[
                start:end
            ] = pred

    return out

# --------------------------------------------------------------------------------------------------
# 5. Allocate final scenario outputs
# --------------------------------------------------------------------------------------------------
det_matrix = np.empty(
    (
        EXPECTED_SCENARIOS,
        EXPECTED_WINDOWS,
    ),
    dtype=np.float64,
)

mc_mean_matrix = np.empty_like(
    det_matrix
)

mc_std_matrix = np.empty_like(
    det_matrix
)

mc_runtime_seconds = np.empty(
    EXPECTED_SCENARIOS,
    dtype=np.float64,
)

scenario_rows = []

# Historical MC helper restores the mode it sees at entry.
model.eval()

run_start = time.time()

# --------------------------------------------------------------------------------------------------
# 6. Full 315-scenario deterministic + 50-pass MC execution
# --------------------------------------------------------------------------------------------------
for i in range(
    EXPECTED_SCENARIOS
):

    row = manifest.iloc[
        i
    ]

    sensor = str(
        row["sensor"]
    )

    try:
        feature_idx = (
            model_features.index(
                sensor
            )
        )

    except ValueError as exc:
        raise RuntimeError(
            f"Scenario {i}: sensor "
            f"{sensor!r} not found in "
            "locked model features."
        ) from exc

    # ----------------------------------------------------------------------
    # Reconstruct full 17-channel corrupted tensor from:
    # CLEAN DEV tensor + exact target-sensor patch from A6-R1.
    # ----------------------------------------------------------------------
    X = clean_x.copy()

    X[
        :,
        :,
        feature_idx,
    ] = patch_bank[
        i
    ]

    if X.shape != (
        EXPECTED_WINDOWS,
        WINDOW_LENGTH,
        FEATURE_COUNT,
    ):
        raise RuntimeError(
            f"Scenario {i}: reconstructed tensor "
            f"shape {X.shape} is invalid."
        )

    if not np.isfinite(
        X
    ).all():
        raise RuntimeError(
            f"Scenario {i}: reconstructed tensor "
            "contains non-finite values."
        )

    # ----------------------------------------------------------------------
    # Deterministic inference
    # ----------------------------------------------------------------------
    det_pred = (
        deterministic_predict_628hc6a8a7r1(
            X
        )
    )

    if det_pred.shape != (
        EXPECTED_WINDOWS,
    ):
        raise RuntimeError(
            f"Scenario {i}: deterministic prediction "
            f"shape {det_pred.shape}; expected "
            f"{(EXPECTED_WINDOWS,)}."
        )

    if not np.isfinite(
        det_pred
    ).all():
        raise RuntimeError(
            f"Scenario {i}: deterministic predictions "
            "contain non-finite values."
        )

    # ----------------------------------------------------------------------
    # Exact historical 50-pass MC-dropout helper
    # ----------------------------------------------------------------------
    mc_out = (
        mc_dropout_predict_628hc0(
            X,
            model,
            passes=MC_PASSES,
            batch_size=MC_BATCH_SIZE,
            seed=MC_SEED,
        )
    )

    # ----------------------------------------------------------------------
    # IMPORTANT:
    #
    # Historical helper contract is EXACTLY:
    #
    #     (mc_mean, mc_std, runtime)
    #
    # NOT a 2-value tuple.
    # ----------------------------------------------------------------------
    if (
        not isinstance(
            mc_out,
            (
                tuple,
                list,
            ),
        )
        or len(
            mc_out
        ) != 3
    ):

        observed_len = (
            len(
                mc_out
            )
            if isinstance(
                mc_out,
                (
                    tuple,
                    list,
                ),
            )
            else "n/a"
        )

        raise RuntimeError(
            f"Scenario {i}: MC helper contract mismatch. "
            f"Expected (mean, std, runtime); "
            f"got type={type(mc_out).__name__}, "
            f"len={observed_len}."
        )

    mc_mean = np.asarray(
        mc_out[
            0
        ],
        dtype=np.float64,
    ).reshape(
        -1
    )

    mc_std = np.asarray(
        mc_out[
            1
        ],
        dtype=np.float64,
    ).reshape(
        -1
    )

    mc_runtime = float(
        mc_out[
            2
        ]
    )

    if mc_mean.shape != (
        EXPECTED_WINDOWS,
    ):
        raise RuntimeError(
            f"Scenario {i}: MC mean shape "
            f"{mc_mean.shape}; expected "
            f"{(EXPECTED_WINDOWS,)}."
        )

    if mc_std.shape != (
        EXPECTED_WINDOWS,
    ):
        raise RuntimeError(
            f"Scenario {i}: MC std shape "
            f"{mc_std.shape}; expected "
            f"{(EXPECTED_WINDOWS,)}."
        )

    if not np.isfinite(
        mc_mean
    ).all():
        raise RuntimeError(
            f"Scenario {i}: MC mean contains "
            "non-finite values."
        )

    if not np.isfinite(
        mc_std
    ).all():
        raise RuntimeError(
            f"Scenario {i}: MC std contains "
            "non-finite values."
        )

    if np.any(
        mc_std < 0.0
    ):
        raise RuntimeError(
            f"Scenario {i}: MC std contains "
            "negative values."
        )

    if (
        not np.isfinite(
            mc_runtime
        )
        or mc_runtime < 0.0
    ):
        raise RuntimeError(
            f"Scenario {i}: invalid MC runtime "
            f"{mc_runtime}."
        )

    # ----------------------------------------------------------------------
    # Store exact outputs
    # ----------------------------------------------------------------------
    det_matrix[
        i
    ] = det_pred

    mc_mean_matrix[
        i
    ] = mc_mean

    mc_std_matrix[
        i
    ] = mc_std

    mc_runtime_seconds[
        i
    ] = mc_runtime

    scenario_rows.append(
        {
            "scenario_index":
                int(i),

            "scenario_number":
                int(
                    row[
                        "scenario_number"
                    ]
                ),

            "scenario_id":
                str(
                    row[
                        "scenario_id"
                    ]
                ),

            "family":
                str(
                    row[
                        "family"
                    ]
                ),

            "severity":
                str(
                    row[
                        "severity"
                    ]
                ),

            "sensor":
                sensor,

            "deterministic_mean":
                float(
                    np.mean(
                        det_pred
                    )
                ),

            "mc_mean_mean":
                float(
                    np.mean(
                        mc_mean
                    )
                ),

            "mc_std_median":
                float(
                    np.median(
                        mc_std
                    )
                ),

            "mc_std_mean":
                float(
                    np.mean(
                        mc_std
                    )
                ),

            "mc_std_max":
                float(
                    np.max(
                        mc_std
                    )
                ),

            "mc_runtime_sec":
                mc_runtime,
        }
    )

    if (
        i == 0
        or (
            i + 1
        ) % 25 == 0
        or (
            i + 1
        ) == EXPECTED_SCENARIOS
    ):

        print(
            f"processed "
            f"{i + 1:>3}/{EXPECTED_SCENARIOS} | "
            f"{str(row['family']):<7} | "
            f"{str(row['severity']):<6} | "
            f"{sensor:<9} | "
            f"MC-STD median="
            f"{np.median(mc_std):.6f}"
        )

total_runtime = float(
    time.time()
    -
    run_start
)

# --------------------------------------------------------------------------------------------------
# 7. Restore original model training/eval state
# --------------------------------------------------------------------------------------------------
if original_training_state:
    model.train()

else:
    model.eval()

model_sha_after = (
    model_state_sha256_628hc6a8a7r1(
        model
    )
)

# --------------------------------------------------------------------------------------------------
# 8. Freeze in-memory outputs for downstream C6-A8 evaluation
# --------------------------------------------------------------------------------------------------
FD001_CORRUPTED_DETERMINISTIC_628HC6A8A7R1 = (
    det_matrix
)

FD001_CORRUPTED_MC_MEAN_628HC6A8A7R1 = (
    mc_mean_matrix
)

FD001_CORRUPTED_MC_STD_628HC6A8A7R1 = (
    mc_std_matrix
)

FD001_CORRUPTED_MC_RUNTIME_628HC6A8A7R1 = (
    mc_runtime_seconds
)

FD001_CORRUPTED_MC_SUMMARY_628HC6A8A7R1 = (
    pd.DataFrame(
        scenario_rows
    )
)

family_severity_summary = (
    FD001_CORRUPTED_MC_SUMMARY_628HC6A8A7R1
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_index",
            "count",
        ),

        median_scenario_mc_std=(
            "mc_std_median",
            "median",
        ),

        mean_scenario_mc_std=(
            "mc_std_mean",
            "mean",
        ),

        max_scenario_mc_std=(
            "mc_std_max",
            "max",
        ),

        median_mc_runtime_sec=(
            "mc_runtime_sec",
            "median",
        ),
    )
)

FD001_CORRUPTED_MC_FAMILY_SEVERITY_628HC6A8A7R1 = (
    family_severity_summary
)

# --------------------------------------------------------------------------------------------------
# 9. Audit
# --------------------------------------------------------------------------------------------------
audit_rows = [

    (
        "manifest_shape_315x9",
        manifest.shape
        ==
        (
            315,
            9,
        ),
    ),

    (
        "clean_dev_shape_3272x30x17",
        clean_x.shape
        ==
        (
            3272,
            30,
            17,
        ),
    ),

    (
        "patch_bank_shape_315x3272x30",
        patch_bank.shape
        ==
        (
            315,
            3272,
            30,
        ),
    ),

    (
        "deterministic_shape_315x3272",
        det_matrix.shape
        ==
        (
            315,
            3272,
        ),
    ),

    (
        "mc_mean_shape_315x3272",
        mc_mean_matrix.shape
        ==
        (
            315,
            3272,
        ),
    ),

    (
        "mc_std_shape_315x3272",
        mc_std_matrix.shape
        ==
        (
            315,
            3272,
        ),
    ),

    (
        "mc_runtime_shape_315",
        mc_runtime_seconds.shape
        ==
        (
            315,
        ),
    ),

    (
        "deterministic_finite",
        bool(
            np.isfinite(
                det_matrix
            ).all()
        ),
    ),

    (
        "mc_mean_finite",
        bool(
            np.isfinite(
                mc_mean_matrix
            ).all()
        ),
    ),

    (
        "mc_std_finite",
        bool(
            np.isfinite(
                mc_std_matrix
            ).all()
        ),
    ),

    (
        "mc_std_nonnegative",
        bool(
            np.all(
                mc_std_matrix
                >=
                0.0
            )
        ),
    ),

    (
        "mc_runtime_finite_nonnegative",
        bool(
            np.isfinite(
                mc_runtime_seconds
            ).all()
            and
            np.all(
                mc_runtime_seconds
                >=
                0.0
            )
        ),
    ),

    (
        "all_315_scenarios_summarized",
        len(
            FD001_CORRUPTED_MC_SUMMARY_628HC6A8A7R1
        )
        ==
        315,
    ),

    (
        "model_state_sha256_unchanged",
        model_sha_before
        ==
        model_sha_after,
    ),

    (
        "model_training_mode_restored",
        bool(
            model.training
        )
        ==
        original_training_state,
    ),

    (
        "test_features_not_read",
        True,
    ),

    (
        "official_test_labels_not_read",
        True,
    ),

    (
        "test_not_scored",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

C6_A8_A7_R1_AUDIT = (
    pd.DataFrame(
        audit_rows,
        columns=[
            "check",
            "passed",
        ],
    )
)

all_checks_passed = bool(
    C6_A8_A7_R1_AUDIT[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------------
# 10. Output summaries
# --------------------------------------------------------------------------------------------------
print(
    "\n--- FAMILY × SEVERITY MC SUMMARY ---"
)

display(
    FD001_CORRUPTED_MC_FAMILY_SEVERITY_628HC6A8A7R1
)

print(
    "\n--- C6-A8-A7-R1 AUDIT ---"
)

display(
    C6_A8_A7_R1_AUDIT
)

print(
    "\n"
    +
    "=" * 118
)

print(
    "STEP 6.28H-C6-A8-A7-R1 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    f"Patch bank source                    : "
    f"{patch_name}"
)

print(
    f"Scenarios processed                  : "
    f"{EXPECTED_SCENARIOS}"
)

print(
    f"Windows per scenario                 : "
    f"{EXPECTED_WINDOWS}"
)

print(
    f"MC passes                            : "
    f"{MC_PASSES}"
)

print(
    f"MC batch size                        : "
    f"{MC_BATCH_SIZE}"
)

print(
    f"MC seed                              : "
    f"{MC_SEED}"
)

print(
    f"Deterministic matrix shape           : "
    f"{det_matrix.shape}"
)

print(
    f"MC mean matrix shape                 : "
    f"{mc_mean_matrix.shape}"
)

print(
    f"MC std matrix shape                  : "
    f"{mc_std_matrix.shape}"
)

print(
    f"Overall MC-STD median                : "
    f"{float(np.median(mc_std_matrix))}"
)

print(
    f"Overall MC-STD mean                  : "
    f"{float(np.mean(mc_std_matrix))}"
)

print(
    f"Overall MC-STD max                   : "
    f"{float(np.max(mc_std_matrix))}"
)

print(
    f"Sum helper-reported MC runtime (sec) : "
    f"{float(np.sum(mc_runtime_seconds))}"
)

print(
    f"Total cell execution runtime (sec)   : "
    f"{total_runtime}"
)

print(
    f"Model-state SHA-256 unchanged        : "
    f"{model_sha_before == model_sha_after}"
)

print(
    "FD001 TEST features read             : False"
)

print(
    "Official FD001 TEST labels read      : False"
)

print(
    "TEST scored                          : False"
)

print(
    "Model retrained                      : False"
)

print(
    "Threshold modified                   : False"
)

print(
    "Files written                        : False"
)

print(
    f"All C6-A8-A7-R1 checks passed        : "
    f"{all_checks_passed}"
)

if not all_checks_passed:

    failed = (
        C6_A8_A7_R1_AUDIT.loc[
            ~C6_A8_A7_R1_AUDIT[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A8-A7-R1 audit failed: "
        +
        ", ".join(
            failed
        )
    )

print(
    "\nPASS — full 315-scenario deterministic + "
    "50-pass MC-dropout execution completed "
    "with the corrected historical helper contract."
)

print(
    "Next: calibrate these corrupted MC uncertainties "
    "with the already-frozen FD001 reliability construction "
    "and evaluate the already-frozen nominal-0.80 policy."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A7-R1 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C6-A8-A8-A0 — FROZEN FD001 CALIBRATION OBJECT LOCK (NO MODEL EXECUTION)
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A8-A8-A0 — FROZEN FD001 CALIBRATION OBJECT LOCK")
print("=" * 118)

snapshot = list(globals().items())

rows = []

for name, obj in snapshot:
    name_upper = name.upper()

    relevant = (
        "628HC6A7A2A2" in name_upper
        or (
            "FD001" in name_upper
            and any(
                token in name_upper
                for token in (
                    "PHASE",
                    "G1",
                    "G2",
                    "THRESHOLD",
                    "FINAL_RISK",
                    "CLEAN_DEV",
                )
            )
        )
    )

    if not relevant:
        continue

    row = {
        "name": name,
        "type": type(obj).__name__,
        "shape_or_len": None,
        "finite": None,
        "min": np.nan,
        "median": np.nan,
        "mean": np.nan,
        "max": np.nan,
    }

    if isinstance(obj, np.ndarray):
        arr = np.asarray(obj)

        row["shape_or_len"] = str(arr.shape)

        if np.issubdtype(arr.dtype, np.number) and arr.size > 0:
            arr64 = np.asarray(arr, dtype=np.float64)
            row["finite"] = bool(np.isfinite(arr64).all())

            if row["finite"]:
                row["min"] = float(np.min(arr64))
                row["median"] = float(np.median(arr64))
                row["mean"] = float(np.mean(arr64))
                row["max"] = float(np.max(arr64))

    elif isinstance(obj, dict):
        row["shape_or_len"] = f"dict[{len(obj)}] keys={list(obj.keys())}"

    elif isinstance(obj, pd.DataFrame):
        row["shape_or_len"] = str(obj.shape)

    elif np.isscalar(obj) and isinstance(
        obj,
        (int, float, np.integer, np.floating),
    ):
        value = float(obj)
        row["shape_or_len"] = "scalar"
        row["finite"] = bool(np.isfinite(value))
        row["min"] = value
        row["median"] = value
        row["mean"] = value
        row["max"] = value

    rows.append(row)

calibration_candidates = (
    pd.DataFrame(rows)
    .sort_values("name")
    .reset_index(drop=True)
)

print("\n--- CALIBRATION / THRESHOLD CANDIDATES ---")
display(calibration_candidates)

target_rows = []

for name, obj in snapshot:
    name_upper = name.upper()

    if not isinstance(obj, np.ndarray):
        continue

    arr = np.asarray(obj)

    if arr.shape != (3272,):
        continue

    if not np.issubdtype(arr.dtype, np.number):
        continue

    arr64 = np.asarray(arr, dtype=np.float64)

    if not np.isfinite(arr64).all():
        continue

    if not (
        np.min(arr64) >= 0.0
        and np.max(arr64) <= 125.0
    ):
        continue

    if not any(
        token in name_upper
        for token in (
            "Y_DEV",
            "TARGET",
            "RUL",
        )
    ):
        continue

    target_rows.append(
        {
            "name": name,
            "shape": str(arr.shape),
            "min": float(np.min(arr64)),
            "median": float(np.median(arr64)),
            "mean": float(np.mean(arr64)),
            "max": float(np.max(arr64)),
        }
    )

target_candidates = (
    pd.DataFrame(target_rows)
    .sort_values("name")
    .reset_index(drop=True)
)

print("\n--- DEV TARGET CANDIDATES ---")
display(target_candidates)

metadata_name = (
    "FD001_DEV_WINDOW_METADATA_628HC6A5R1"
    if "FD001_DEV_WINDOW_METADATA_628HC6A5R1" in globals()
    else None
)

print("\n--- LOCK SUMMARY ---")
print(
    "Calibration candidates found :",
    len(calibration_candidates),
)
print(
    "DEV target candidates found  :",
    len(target_candidates),
)
print(
    "DEV metadata exact object     :",
    metadata_name,
)
print("Model executed                : False")
print("MC dropout executed           : False")
print("Model retrained               : False")
print("TEST features read            : False")
print("Official TEST labels read     : False")
print("Threshold modified            : False")
print("Files written                 : False")

print(
    "\nSend this output back. "
    "The next cell will use only the exact frozen calibration objects identified here."
)

print("\n=== END STEP 6.28H-C6-A8-A8-A0 ===")

In [ ]:
# ==================================================================================================
# STEP 6.28H-C6-A8-A8-A1 — FD001 CORRUPTED RELIABILITY CALIBRATION + FROZEN 0.80 POLICY EVALUATION
# Reuses completed A7-R1 deterministic/MC outputs.
# NO model execution. NO MC dropout. NO threshold tuning.
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C6-A8-A8-A1 — FD001 CORRUPTED RELIABILITY CALIBRATION + FROZEN 0.80 POLICY EVALUATION")
print("=" * 122)

# --------------------------------------------------------------------------------------------------
# 0. Exact locked-object contract
# --------------------------------------------------------------------------------------------------
required_names = [
    "FD001_CORRUPTED_DETERMINISTIC_628HC6A8A7R1",
    "FD001_CORRUPTED_MC_STD_628HC6A8A7R1",
    "FD001_CORRUPTION_MANIFEST_628HC6A8A3",
    "FD001_Y_DEV_WINDOWS_628HC6A5R1",
    "FD001_DEV_WINDOW_METADATA_628HC6A5R1",
    "FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2",
    "FD001_DEV_G1_RISK_628HC6A7A2A2",
    "FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2",
    "FD001_DEV_ACCEPT_MASK_628HC6A7A2A2",
    "FD001_DEV_ABSTAIN_MASK_628HC6A7A2A2",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "fd001_phase_adjust_628hc6a7a2a2",
    "fd001_g1_risk_628hc6a7a2a2",
    "fd001_final_risk_628hc6a7a2a2",
    "dev_pred_628hc6a7a2a2",
    "dev_std_628hc6a7a2a2",
    "C6_A8_A7_R1_AUDIT",
]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required locked object(s): "
        + ", ".join(missing)
    )

if not bool(C6_A8_A7_R1_AUDIT["passed"].all()):
    raise RuntimeError(
        "C6-A8-A7-R1 audit is not fully passed."
    )

# --------------------------------------------------------------------------------------------------
# 1. Exact frozen inputs
# --------------------------------------------------------------------------------------------------
EXPECTED_SCENARIOS = 315
EXPECTED_WINDOWS = 3272

det_matrix = np.asarray(
    FD001_CORRUPTED_DETERMINISTIC_628HC6A8A7R1,
    dtype=np.float64,
)

mc_std_matrix = np.asarray(
    FD001_CORRUPTED_MC_STD_628HC6A8A7R1,
    dtype=np.float64,
)

manifest = (
    FD001_CORRUPTION_MANIFEST_628HC6A8A3
    .reset_index(drop=True)
    .copy()
)

y_dev = np.asarray(
    FD001_Y_DEV_WINDOWS_628HC6A5R1,
    dtype=np.float64,
).reshape(-1)

dev_meta = (
    FD001_DEV_WINDOW_METADATA_628HC6A5R1
    .copy()
)

clean_pred = np.asarray(
    dev_pred_628hc6a7a2a2,
    dtype=np.float64,
).reshape(-1)

clean_std = np.asarray(
    dev_std_628hc6a7a2a2,
    dtype=np.float64,
).reshape(-1)

clean_phase_frozen = np.asarray(
    FD001_DEV_PHASE_ADJUSTED_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1)

clean_g1_frozen = np.asarray(
    FD001_DEV_G1_RISK_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1)

clean_risk_frozen = np.asarray(
    FD001_DEV_UNCERTAINTY_RISK_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1)

clean_accept_frozen = np.asarray(
    FD001_DEV_ACCEPT_MASK_628HC6A7A2A2,
    dtype=bool,
).reshape(-1)

clean_abstain_frozen = np.asarray(
    FD001_DEV_ABSTAIN_MASK_628HC6A7A2A2,
    dtype=bool,
).reshape(-1)

threshold_before = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

# --------------------------------------------------------------------------------------------------
# 2. Shape / finite guards
# --------------------------------------------------------------------------------------------------
if det_matrix.shape != (315, 3272):
    raise RuntimeError(
        f"Deterministic matrix shape {det_matrix.shape}; "
        "expected (315, 3272)."
    )

if mc_std_matrix.shape != (315, 3272):
    raise RuntimeError(
        f"MC-STD matrix shape {mc_std_matrix.shape}; "
        "expected (315, 3272)."
    )

if manifest.shape != (315, 9):
    raise RuntimeError(
        f"Manifest shape {manifest.shape}; "
        "expected (315, 9)."
    )

if y_dev.shape != (3272,):
    raise RuntimeError(
        f"DEV target shape {y_dev.shape}; "
        "expected (3272,)."
    )

if dev_meta.shape[0] != 3272:
    raise RuntimeError(
        f"DEV metadata rows {dev_meta.shape[0]}; "
        "expected 3272."
    )

for name, arr in [
    ("deterministic matrix", det_matrix),
    ("MC-STD matrix", mc_std_matrix),
    ("DEV target", y_dev),
    ("clean prediction", clean_pred),
    ("clean MC-STD", clean_std),
    ("frozen clean phase", clean_phase_frozen),
    ("frozen clean G1", clean_g1_frozen),
    ("frozen clean risk", clean_risk_frozen),
]:
    if not np.isfinite(arr).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )

if np.any(mc_std_matrix < 0.0):
    raise RuntimeError(
        "Corrupted MC-STD matrix contains negative values."
    )

if np.any(clean_std < 0.0):
    raise RuntimeError(
        "Clean MC-STD contains negative values."
    )

# --------------------------------------------------------------------------------------------------
# 3. CLEAN-DEV calibration identity lock
# --------------------------------------------------------------------------------------------------
clean_phase_rebuilt = np.asarray(
    fd001_phase_adjust_628hc6a7a2a2(
        clean_pred,
        clean_std,
    ),
    dtype=np.float64,
).reshape(-1)

clean_g1_rebuilt = np.asarray(
    fd001_g1_risk_628hc6a7a2a2(
        clean_phase_rebuilt
    ),
    dtype=np.float64,
).reshape(-1)

clean_risk_rebuilt = np.asarray(
    fd001_final_risk_628hc6a7a2a2(
        clean_g1_rebuilt
    ),
    dtype=np.float64,
).reshape(-1)

phase_max_error = float(
    np.max(
        np.abs(
            clean_phase_rebuilt
            -
            clean_phase_frozen
        )
    )
)

g1_max_error = float(
    np.max(
        np.abs(
            clean_g1_rebuilt
            -
            clean_g1_frozen
        )
    )
)

risk_max_error = float(
    np.max(
        np.abs(
            clean_risk_rebuilt
            -
            clean_risk_frozen
        )
    )
)

IDENTITY_ATOL = 1e-14

if not (
    phase_max_error <= IDENTITY_ATOL
    and g1_max_error <= IDENTITY_ATOL
    and risk_max_error <= IDENTITY_ATOL
):
    raise RuntimeError(
        "Frozen CLEAN-DEV calibration identity failed. "
        f"phase={phase_max_error:.3e}, "
        f"G1={g1_max_error:.3e}, "
        f"risk={risk_max_error:.3e}."
    )

# --------------------------------------------------------------------------------------------------
# 4. Recover exact frozen threshold comparator
#    This does NOT tune the threshold.
# --------------------------------------------------------------------------------------------------
mask_le = (
    clean_risk_frozen
    <=
    threshold_before
)

mask_lt = (
    clean_risk_frozen
    <
    threshold_before
)

le_matches = bool(
    np.array_equal(
        mask_le,
        clean_accept_frozen,
    )
)

lt_matches = bool(
    np.array_equal(
        mask_lt,
        clean_accept_frozen,
    )
)

if le_matches and not lt_matches:
    threshold_comparator = "<="

elif lt_matches and not le_matches:
    threshold_comparator = "<"

elif le_matches and lt_matches:
    threshold_comparator = "<="

else:
    raise RuntimeError(
        "Frozen CLEAN accept-mask semantics cannot be "
        "reproduced by either <= or <."
    )

if not np.array_equal(
    ~clean_accept_frozen,
    clean_abstain_frozen,
):
    raise RuntimeError(
        "Frozen accept and abstain masks are not exact complements."
    )

def frozen_accept_mask_628hc6a8a8a1(risk_values):

    risk_values = np.asarray(
        risk_values,
        dtype=np.float64,
    )

    if threshold_comparator == "<=":
        return (
            risk_values
            <=
            threshold_before
        )

    return (
        risk_values
        <
        threshold_before
    )

# --------------------------------------------------------------------------------------------------
# 5. CLEAN DEV frozen-policy reference
# --------------------------------------------------------------------------------------------------
clean_accept = (
    frozen_accept_mask_628hc6a8a8a1(
        clean_risk_frozen
    )
)

if not np.array_equal(
    clean_accept,
    clean_accept_frozen,
):
    raise RuntimeError(
        "Recovered comparator does not reproduce "
        "the frozen CLEAN accept mask."
    )

clean_abs_error = np.abs(
    clean_pred
    -
    y_dev
)

clean_accepted_count = int(
    np.count_nonzero(
        clean_accept
    )
)

clean_abstained_count = int(
    EXPECTED_WINDOWS
    -
    clean_accepted_count
)

clean_coverage = float(
    clean_accepted_count
    /
    EXPECTED_WINDOWS
)

clean_full_mae = float(
    np.mean(
        clean_abs_error
    )
)

clean_accepted_mae = float(
    np.mean(
        clean_abs_error[
            clean_accept
        ]
    )
)

clean_abstained_mae = float(
    np.mean(
        clean_abs_error[
            ~clean_accept
        ]
    )
)

clean_selective_gain = float(
    clean_full_mae
    -
    clean_accepted_mae
)

FD001_CLEAN_POLICY_REFERENCE_628HC6A8A8A1 = pd.DataFrame(
    [
        {
            "population": "clean_dev",
            "windows": EXPECTED_WINDOWS,
            "accepted": clean_accepted_count,
            "abstained": clean_abstained_count,
            "coverage": clean_coverage,
            "full_mae": clean_full_mae,
            "accepted_mae": clean_accepted_mae,
            "abstained_mae": clean_abstained_mae,
            "selective_gain_mae": clean_selective_gain,
            "risk_median": float(
                np.median(
                    clean_risk_frozen
                )
            ),
            "risk_mean": float(
                np.mean(
                    clean_risk_frozen
                )
            ),
            "risk_max": float(
                np.max(
                    clean_risk_frozen
                )
            ),
        }
    ]
)

# --------------------------------------------------------------------------------------------------
# 6. Calibrate all 315 already-computed corrupted scenarios
# --------------------------------------------------------------------------------------------------
phase_matrix = np.empty_like(
    det_matrix
)

g1_matrix = np.empty_like(
    det_matrix
)

risk_matrix = np.empty_like(
    det_matrix
)

scenario_rows = []

for i in range(EXPECTED_SCENARIOS):

    det_i = det_matrix[i]
    std_i = mc_std_matrix[i]

    phase_i = np.asarray(
        fd001_phase_adjust_628hc6a7a2a2(
            det_i,
            std_i,
        ),
        dtype=np.float64,
    ).reshape(-1)

    g1_i = np.asarray(
        fd001_g1_risk_628hc6a7a2a2(
            phase_i
        ),
        dtype=np.float64,
    ).reshape(-1)

    risk_i = np.asarray(
        fd001_final_risk_628hc6a7a2a2(
            g1_i
        ),
        dtype=np.float64,
    ).reshape(-1)

    if (
        phase_i.shape != (3272,)
        or g1_i.shape != (3272,)
        or risk_i.shape != (3272,)
    ):
        raise RuntimeError(
            f"Scenario {i}: calibrated vector shape mismatch."
        )

    if not (
        np.isfinite(
            phase_i
        ).all()
        and np.isfinite(
            g1_i
        ).all()
        and np.isfinite(
            risk_i
        ).all()
    ):
        raise RuntimeError(
            f"Scenario {i}: calibration produced non-finite values."
        )

    if (
        np.min(
            phase_i
        ) < 0.0
        or np.max(
            phase_i
        ) > 1.0
        or np.min(
            g1_i
        ) < 0.0
        or np.max(
            g1_i
        ) > 1.0
        or np.min(
            risk_i
        ) < 0.0
        or np.max(
            risk_i
        ) > 1.0
    ):
        raise RuntimeError(
            f"Scenario {i}: calibrated value escaped [0,1]."
        )

    phase_matrix[i] = phase_i
    g1_matrix[i] = g1_i
    risk_matrix[i] = risk_i

    accept_i = (
        frozen_accept_mask_628hc6a8a8a1(
            risk_i
        )
    )

    accepted_i = int(
        np.count_nonzero(
            accept_i
        )
    )

    abstained_i = int(
        EXPECTED_WINDOWS
        -
        accepted_i
    )

    coverage_i = float(
        accepted_i
        /
        EXPECTED_WINDOWS
    )

    abs_error_i = np.abs(
        det_i
        -
        y_dev
    )

    full_mae_i = float(
        np.mean(
            abs_error_i
        )
    )

    if accepted_i > 0:

        accepted_mae_i = float(
            np.mean(
                abs_error_i[
                    accept_i
                ]
            )
        )

    else:

        accepted_mae_i = np.nan

    if abstained_i > 0:

        abstained_mae_i = float(
            np.mean(
                abs_error_i[
                    ~accept_i
                ]
            )
        )

    else:

        abstained_mae_i = np.nan

    if np.isfinite(
        accepted_mae_i
    ):

        selective_gain_i = float(
            full_mae_i
            -
            accepted_mae_i
        )

    else:

        selective_gain_i = np.nan

    row = manifest.iloc[i]

    scenario_rows.append(
        {
            "scenario_index": int(i),
            "scenario_number": int(
                row["scenario_number"]
            ),
            "scenario_id": str(
                row["scenario_id"]
            ),
            "family": str(
                row["family"]
            ),
            "severity": str(
                row["severity"]
            ),
            "sensor": str(
                row["sensor"]
            ),
            "accepted": accepted_i,
            "abstained": abstained_i,
            "coverage": coverage_i,
            "coverage_delta_vs_clean": float(
                coverage_i
                -
                clean_coverage
            ),
            "full_mae": full_mae_i,
            "accepted_mae": accepted_mae_i,
            "abstained_mae": abstained_mae_i,
            "selective_gain_mae": selective_gain_i,
            "risk_median": float(
                np.median(
                    risk_i
                )
            ),
            "risk_mean": float(
                np.mean(
                    risk_i
                )
            ),
            "risk_max": float(
                np.max(
                    risk_i
                )
            ),
        }
    )

# --------------------------------------------------------------------------------------------------
# 7. Freeze calibrated corrupted outputs in memory
# --------------------------------------------------------------------------------------------------
FD001_CORRUPTED_PHASE_ADJUSTED_628HC6A8A8A1 = (
    phase_matrix
)

FD001_CORRUPTED_G1_RISK_628HC6A8A8A1 = (
    g1_matrix
)

FD001_CORRUPTED_UNCERTAINTY_RISK_628HC6A8A8A1 = (
    risk_matrix
)

FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1 = (
    pd.DataFrame(
        scenario_rows
    )
)

# --------------------------------------------------------------------------------------------------
# 8. Family × severity summary
# --------------------------------------------------------------------------------------------------
def q10_finite_628hc6a8a8a1(values):

    x = np.asarray(
        values,
        dtype=np.float64,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if x.size == 0:
        return np.nan

    return float(
        np.quantile(
            x,
            0.10,
        )
    )


def count_positive_finite_628hc6a8a8a1(values):

    x = np.asarray(
        values,
        dtype=np.float64,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    return int(
        np.count_nonzero(
            x > 0.0
        )
    )


FD001_CORRUPTED_POLICY_FAMILY_SEVERITY_628HC6A8A8A1 = (
    FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_index",
            "count",
        ),
        median_coverage=(
            "coverage",
            "median",
        ),
        min_coverage=(
            "coverage",
            "min",
        ),
        max_coverage=(
            "coverage",
            "max",
        ),
        median_full_mae=(
            "full_mae",
            "median",
        ),
        median_accepted_mae=(
            "accepted_mae",
            "median",
        ),
        median_selective_gain_mae=(
            "selective_gain_mae",
            "median",
        ),
        mean_selective_gain_mae=(
            "selective_gain_mae",
            "mean",
        ),
        q10_selective_gain_mae=(
            "selective_gain_mae",
            q10_finite_628hc6a8a8a1,
        ),
        positive_gain_scenarios=(
            "selective_gain_mae",
            count_positive_finite_628hc6a8a8a1,
        ),
        median_risk=(
            "risk_median",
            "median",
        ),
    )
)

# --------------------------------------------------------------------------------------------------
# 9. Diagnostic worst scenarios
# --------------------------------------------------------------------------------------------------
FD001_CORRUPTED_POLICY_WORST15_628HC6A8A8A1 = (
    FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1
    .sort_values(
        [
            "selective_gain_mae",
            "accepted_mae",
        ],
        ascending=[
            True,
            False,
        ],
        na_position="first",
    )
    .head(
        15
    )
    .reset_index(
        drop=True
    )
)

# --------------------------------------------------------------------------------------------------
# 10. Audit
# --------------------------------------------------------------------------------------------------
threshold_after = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

audit_rows = [
    (
        "previous_A7_R1_passed",
        bool(
            C6_A8_A7_R1_AUDIT[
                "passed"
            ].all()
        ),
    ),
    (
        "corrupted_deterministic_shape_315x3272",
        det_matrix.shape == (315, 3272),
    ),
    (
        "corrupted_mc_std_shape_315x3272",
        mc_std_matrix.shape == (315, 3272),
    ),
    (
        "clean_phase_identity_within_1e14",
        phase_max_error <= IDENTITY_ATOL,
    ),
    (
        "clean_g1_identity_within_1e14",
        g1_max_error <= IDENTITY_ATOL,
    ),
    (
        "clean_final_risk_identity_within_1e14",
        risk_max_error <= IDENTITY_ATOL,
    ),
    (
        "frozen_threshold_comparator_recovered",
        threshold_comparator in {
            "<=",
            "<",
        },
    ),
    (
        "clean_accept_mask_reproduced_exactly",
        np.array_equal(
            clean_accept,
            clean_accept_frozen,
        ),
    ),
    (
        "clean_abstain_mask_is_complement",
        np.array_equal(
            ~clean_accept,
            clean_abstain_frozen,
        ),
    ),
    (
        "clean_accepted_count_2619",
        clean_accepted_count == 2619,
    ),
    (
        "phase_matrix_shape_315x3272",
        phase_matrix.shape == (315, 3272),
    ),
    (
        "g1_matrix_shape_315x3272",
        g1_matrix.shape == (315, 3272),
    ),
    (
        "risk_matrix_shape_315x3272",
        risk_matrix.shape == (315, 3272),
    ),
    (
        "phase_matrix_finite",
        bool(
            np.isfinite(
                phase_matrix
            ).all()
        ),
    ),
    (
        "g1_matrix_finite",
        bool(
            np.isfinite(
                g1_matrix
            ).all()
        ),
    ),
    (
        "risk_matrix_finite",
        bool(
            np.isfinite(
                risk_matrix
            ).all()
        ),
    ),
    (
        "phase_matrix_in_0_1",
        bool(
            np.min(
                phase_matrix
            ) >= 0.0
            and np.max(
                phase_matrix
            ) <= 1.0
        ),
    ),
    (
        "g1_matrix_in_0_1",
        bool(
            np.min(
                g1_matrix
            ) >= 0.0
            and np.max(
                g1_matrix
            ) <= 1.0
        ),
    ),
    (
        "risk_matrix_in_0_1",
        bool(
            np.min(
                risk_matrix
            ) >= 0.0
            and np.max(
                risk_matrix
            ) <= 1.0
        ),
    ),
    (
        "all_315_policy_rows_present",
        len(
            FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1
        ) == 315,
    ),
    (
        "all_scenario_ids_unique",
        FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
            "scenario_id"
        ].nunique() == 315,
    ),
    (
        "threshold_unchanged",
        threshold_before
        ==
        threshold_after,
    ),
    (
        "test_features_not_read",
        True,
    ),
    (
        "official_test_labels_not_read",
        True,
    ),
    (
        "test_not_scored",
        True,
    ),
    (
        "no_model_execution_in_this_step",
        True,
    ),
    (
        "no_mc_dropout_execution_in_this_step",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning_or_modification",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

C6_A8_A8_A1_AUDIT = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)

all_checks_passed = bool(
    C6_A8_A8_A1_AUDIT[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------------
# 11. Output
# --------------------------------------------------------------------------------------------------
print(
    "\n--- FROZEN CLEAN-DEV IDENTITY ---"
)

print(
    "Phase-adjusted max abs error :",
    f"{phase_max_error:.3e}",
)

print(
    "G1-risk max abs error        :",
    f"{g1_max_error:.3e}",
)

print(
    "Final-risk max abs error     :",
    f"{risk_max_error:.3e}",
)

print(
    "Threshold comparator         :",
    f"risk {threshold_comparator} threshold",
)

print(
    "\n--- CLEAN DEV FROZEN POLICY ---"
)

display(
    FD001_CLEAN_POLICY_REFERENCE_628HC6A8A8A1
)

print(
    "\n--- FAMILY × SEVERITY FROZEN-POLICY SUMMARY ---"
)

display(
    FD001_CORRUPTED_POLICY_FAMILY_SEVERITY_628HC6A8A8A1
)

print(
    "\n--- 15 WORST SCENARIOS BY SELECTIVE GAIN ---"
)

display(
    FD001_CORRUPTED_POLICY_WORST15_628HC6A8A8A1
)

print(
    "\n--- C6-A8-A8-A1 AUDIT ---"
)

display(
    C6_A8_A8_A1_AUDIT
)

print("\n" + "=" * 122)
print("STEP 6.28H-C6-A8-A8-A1 FINAL SUMMARY")
print("=" * 122)

print(
    "Frozen FD001 threshold                  :",
    threshold_before,
)

print(
    "Threshold comparator                    :",
    f"risk {threshold_comparator} threshold",
)

print(
    "Nominal coverage                        :",
    0.80,
)

print(
    "CLEAN DEV realized coverage             :",
    clean_coverage,
)

print(
    "CLEAN DEV full MAE                      :",
    clean_full_mae,
)

print(
    "CLEAN DEV accepted MAE                  :",
    clean_accepted_mae,
)

print(
    "CLEAN DEV selective gain (MAE)          :",
    clean_selective_gain,
)

print(
    "Corrupted risk matrix shape             :",
    risk_matrix.shape,
)

print(
    "Median corrupted-scenario coverage      :",
    float(
        FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
            "coverage"
        ].median()
    ),
)

print(
    "Median corrupted-scenario full MAE      :",
    float(
        FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
            "full_mae"
        ].median()
    ),
)

print(
    "Median corrupted-scenario accepted MAE  :",
    float(
        FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
            "accepted_mae"
        ].median()
    ),
)

print(
    "Median corrupted selective gain (MAE)   :",
    float(
        FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
            "selective_gain_mae"
        ].median()
    ),
)

print(
    "Positive-gain corrupted scenarios       :",
    int(
        (
            FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1[
                "selective_gain_mae"
            ]
            >
            0.0
        ).sum()
    ),
    "/315",
)

print(
    "Model executed in this step             : False"
)

print(
    "MC dropout executed in this step        : False"
)

print(
    "Model retrained                         : False"
)

print(
    "FD001 TEST features read                : False"
)

print(
    "Official FD001 TEST labels read         : False"
)

print(
    "TEST scored                             : False"
)

print(
    "Threshold tuned/modified                : False"
)

print(
    "Files written                           : False"
)

print(
    "All C6-A8-A8-A1 checks passed           :",
    all_checks_passed,
)

if not all_checks_passed:

    failed = C6_A8_A8_A1_AUDIT.loc[
        ~C6_A8_A8_A1_AUDIT[
            "passed"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "C6-A8-A8-A1 audit failed: "
        +
        ", ".join(
            failed
        )
    )

print(
    "\nPASS — all 315 corrupted scenarios were calibrated "
    "with the already-frozen FD001 reliability construction "
    "and evaluated using the already-frozen nominal-0.80 policy."
)

print(
    "No model/MC rerun occurred, and no threshold was selected or modified."
)

print(
    "Next: inspect robustness/failure modes from these frozen-policy results "
    "before any TEST access."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A8-A1 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C6-A8-A8-A2 — FD001 FROZEN-POLICY ROBUSTNESS / FAILURE-MODE DIAGNOSTIC
# Uses only the already-computed A8-A1 policy results.
# NO model execution. NO MC dropout. NO threshold tuning. NO TEST access.
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C6-A8-A8-A2 — FD001 FROZEN-POLICY ROBUSTNESS / FAILURE-MODE DIAGNOSTIC")
print("=" * 122)

# --------------------------------------------------------------------------------------------------
# 0. Locked inputs
# --------------------------------------------------------------------------------------------------
required_names = [
    "FD001_CLEAN_POLICY_REFERENCE_628HC6A8A8A1",
    "FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1",
    "FD001_CORRUPTED_UNCERTAINTY_RISK_628HC6A8A8A1",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "C6_A8_A8_A1_AUDIT",
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required locked object(s): "
        + ", ".join(missing)
    )

if not bool(
    C6_A8_A8_A1_AUDIT["passed"].all()
):
    raise RuntimeError(
        "C6-A8-A8-A1 must pass before robustness diagnostics."
    )

threshold_before = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

clean_ref = (
    FD001_CLEAN_POLICY_REFERENCE_628HC6A8A8A1
    .copy(deep=True)
)

policy = (
    FD001_CORRUPTED_FROZEN_POLICY_628HC6A8A8A1
    .copy(deep=True)
)

risk_matrix = np.asarray(
    FD001_CORRUPTED_UNCERTAINTY_RISK_628HC6A8A8A1,
    dtype=np.float64,
)

# --------------------------------------------------------------------------------------------------
# 1. Structural contract
# --------------------------------------------------------------------------------------------------
expected_policy_columns = [
    "scenario_index",
    "scenario_number",
    "scenario_id",
    "family",
    "severity",
    "sensor",
    "accepted",
    "abstained",
    "coverage",
    "coverage_delta_vs_clean",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_gain_mae",
    "risk_median",
    "risk_mean",
    "risk_max",
]

missing_columns = [
    c for c in expected_policy_columns
    if c not in policy.columns
]

if missing_columns:
    raise RuntimeError(
        "A8-A1 policy table is missing column(s): "
        + ", ".join(missing_columns)
    )

if policy.shape[0] != 315:
    raise RuntimeError(
        f"Expected 315 scenario rows; found {policy.shape[0]}."
    )

if risk_matrix.shape != (315, 3272):
    raise RuntimeError(
        f"Risk matrix shape {risk_matrix.shape}; "
        "expected (315, 3272)."
    )

# --------------------------------------------------------------------------------------------------
# 2. CLEAN frozen-policy anchor
# --------------------------------------------------------------------------------------------------
clean_row = clean_ref.iloc[0]

clean_coverage = float(
    clean_row["coverage"]
)

clean_full_mae = float(
    clean_row["full_mae"]
)

clean_accepted_mae = float(
    clean_row["accepted_mae"]
)

clean_abstained_mae = float(
    clean_row["abstained_mae"]
)

clean_gain = float(
    clean_row["selective_gain_mae"]
)

# --------------------------------------------------------------------------------------------------
# 3. Scenario-level robustness diagnostics
# --------------------------------------------------------------------------------------------------
policy["coverage_error_vs_nominal"] = (
    policy["coverage"]
    - 0.80
)

policy["abs_coverage_error_vs_nominal"] = np.abs(
    policy["coverage_error_vs_nominal"]
)

policy["full_mae_delta_vs_clean"] = (
    policy["full_mae"]
    - clean_full_mae
)

policy["accepted_mae_delta_vs_clean"] = (
    policy["accepted_mae"]
    - clean_accepted_mae
)

policy["selective_gain_delta_vs_clean"] = (
    policy["selective_gain_mae"]
    - clean_gain
)

policy["gain_retention_ratio_vs_clean"] = (
    policy["selective_gain_mae"]
    / clean_gain
)

policy["abstained_minus_accepted_mae"] = (
    policy["abstained_mae"]
    - policy["accepted_mae"]
)

policy["accepted_better_than_abstained"] = (
    policy["abstained_minus_accepted_mae"]
    > 0.0
)

FD001_CORRUPTED_POLICY_ROBUSTNESS_628HC6A8A8A2 = (
    policy
)

# --------------------------------------------------------------------------------------------------
# 4. Helper diagnostics
# --------------------------------------------------------------------------------------------------
def q10_628hc6a8a8a2(values):
    x = np.asarray(
        values,
        dtype=np.float64,
    )
    x = x[np.isfinite(x)]

    if x.size == 0:
        return np.nan

    return float(
        np.quantile(
            x,
            0.10,
        )
    )


def finite_min_628hc6a8a8a2(values):
    x = np.asarray(
        values,
        dtype=np.float64,
    )
    x = x[np.isfinite(x)]

    if x.size == 0:
        return np.nan

    return float(
        np.min(x)
    )


def positive_count_628hc6a8a8a2(values):
    x = np.asarray(
        values,
        dtype=np.float64,
    )
    x = x[np.isfinite(x)]

    return int(
        np.count_nonzero(
            x > 0.0
        )
    )


def true_count_628hc6a8a8a2(values):
    return int(
        np.count_nonzero(
            np.asarray(
                values,
                dtype=bool,
            )
        )
    )

# --------------------------------------------------------------------------------------------------
# 5. Family × severity diagnostics
# --------------------------------------------------------------------------------------------------
FD001_ROBUSTNESS_FAMILY_SEVERITY_628HC6A8A8A2 = (
    policy
    .groupby(
        [
            "family",
            "severity",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),
        median_coverage=(
            "coverage",
            "median",
        ),
        min_coverage=(
            "coverage",
            "min",
        ),
        max_coverage=(
            "coverage",
            "max",
        ),
        median_abs_coverage_error=(
            "abs_coverage_error_vs_nominal",
            "median",
        ),
        median_full_mae=(
            "full_mae",
            "median",
        ),
        max_full_mae=(
            "full_mae",
            "max",
        ),
        median_accepted_mae=(
            "accepted_mae",
            "median",
        ),
        max_accepted_mae=(
            "accepted_mae",
            "max",
        ),
        median_gain=(
            "selective_gain_mae",
            "median",
        ),
        mean_gain=(
            "selective_gain_mae",
            "mean",
        ),
        q10_gain=(
            "selective_gain_mae",
            q10_628hc6a8a8a2,
        ),
        min_gain=(
            "selective_gain_mae",
            finite_min_628hc6a8a8a2,
        ),
        positive_gain_scenarios=(
            "selective_gain_mae",
            positive_count_628hc6a8a8a2,
        ),
        median_gain_retention=(
            "gain_retention_ratio_vs_clean",
            "median",
        ),
        accepted_better_than_abstained_count=(
            "accepted_better_than_abstained",
            true_count_628hc6a8a8a2,
        ),
    )
)

# --------------------------------------------------------------------------------------------------
# 6. Family-level diagnostics
# --------------------------------------------------------------------------------------------------
FD001_ROBUSTNESS_FAMILY_628HC6A8A8A2 = (
    policy
    .groupby(
        "family",
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),
        median_coverage=(
            "coverage",
            "median",
        ),
        min_coverage=(
            "coverage",
            "min",
        ),
        median_full_mae=(
            "full_mae",
            "median",
        ),
        median_accepted_mae=(
            "accepted_mae",
            "median",
        ),
        median_gain=(
            "selective_gain_mae",
            "median",
        ),
        q10_gain=(
            "selective_gain_mae",
            q10_628hc6a8a8a2,
        ),
        min_gain=(
            "selective_gain_mae",
            finite_min_628hc6a8a8a2,
        ),
        median_gain_retention=(
            "gain_retention_ratio_vs_clean",
            "median",
        ),
        accepted_better_than_abstained_count=(
            "accepted_better_than_abstained",
            true_count_628hc6a8a8a2,
        ),
    )
)

# --------------------------------------------------------------------------------------------------
# 7. Sensor vulnerability diagnostics
# --------------------------------------------------------------------------------------------------
FD001_ROBUSTNESS_SENSOR_628HC6A8A8A2 = (
    policy
    .groupby(
        "sensor",
        as_index=False,
        sort=True,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),
        median_coverage=(
            "coverage",
            "median",
        ),
        min_coverage=(
            "coverage",
            "min",
        ),
        median_full_mae=(
            "full_mae",
            "median",
        ),
        max_full_mae=(
            "full_mae",
            "max",
        ),
        median_accepted_mae=(
            "accepted_mae",
            "median",
        ),
        max_accepted_mae=(
            "accepted_mae",
            "max",
        ),
        median_gain=(
            "selective_gain_mae",
            "median",
        ),
        min_gain=(
            "selective_gain_mae",
            finite_min_628hc6a8a8a2,
        ),
        median_gain_retention=(
            "gain_retention_ratio_vs_clean",
            "median",
        ),
        median_error_separation=(
            "abstained_minus_accepted_mae",
            "median",
        ),
    )
    .sort_values(
        [
            "min_gain",
            "max_accepted_mae",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

# --------------------------------------------------------------------------------------------------
# 8. Global robustness summary
# --------------------------------------------------------------------------------------------------
all_gain_values = np.asarray(
    policy["selective_gain_mae"],
    dtype=np.float64,
)

all_separation_values = np.asarray(
    policy["abstained_minus_accepted_mae"],
    dtype=np.float64,
)

within_1pct_count = int(
    np.count_nonzero(
        policy["abs_coverage_error_vs_nominal"]
        <= 0.01
    )
)

within_2pct_count = int(
    np.count_nonzero(
        policy["abs_coverage_error_vs_nominal"]
        <= 0.02
    )
)

positive_gain_count = int(
    np.count_nonzero(
        all_gain_values > 0.0
    )
)

positive_separation_count = int(
    np.count_nonzero(
        all_separation_values > 0.0
    )
)

FD001_ROBUSTNESS_GLOBAL_628HC6A8A8A2 = pd.DataFrame(
    [
        {
            "clean_coverage":
                clean_coverage,

            "clean_full_mae":
                clean_full_mae,

            "clean_accepted_mae":
                clean_accepted_mae,

            "clean_abstained_mae":
                clean_abstained_mae,

            "clean_selective_gain":
                clean_gain,

            "median_corrupted_coverage":
                float(
                    policy["coverage"].median()
                ),

            "median_abs_coverage_error":
                float(
                    policy[
                        "abs_coverage_error_vs_nominal"
                    ].median()
                ),

            "coverage_within_1pct_count":
                within_1pct_count,

            "coverage_within_2pct_count":
                within_2pct_count,

            "median_corrupted_full_mae":
                float(
                    policy["full_mae"].median()
                ),

            "median_corrupted_accepted_mae":
                float(
                    policy["accepted_mae"].median()
                ),

            "median_selective_gain":
                float(
                    policy[
                        "selective_gain_mae"
                    ].median()
                ),

            "q10_selective_gain":
                float(
                    np.quantile(
                        all_gain_values,
                        0.10,
                    )
                ),

            "min_selective_gain":
                float(
                    np.min(
                        all_gain_values
                    )
                ),

            "positive_gain_scenarios":
                positive_gain_count,

            "positive_error_separation_scenarios":
                positive_separation_count,

            "median_gain_retention_ratio":
                float(
                    policy[
                        "gain_retention_ratio_vs_clean"
                    ].median()
                ),
        }
    ]
)

# --------------------------------------------------------------------------------------------------
# 9. Worst-case diagnostic tables
# --------------------------------------------------------------------------------------------------
worst_columns = [
    "scenario_number",
    "scenario_id",
    "family",
    "severity",
    "sensor",
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_gain_mae",
    "gain_retention_ratio_vs_clean",
    "abstained_minus_accepted_mae",
    "risk_median",
]

FD001_ROBUSTNESS_LOWEST_GAIN20_628HC6A8A8A2 = (
    policy
    .sort_values(
        [
            "selective_gain_mae",
            "accepted_mae",
        ],
        ascending=[
            True,
            False,
        ],
    )
    [worst_columns]
    .head(20)
    .reset_index(drop=True)
)

FD001_ROBUSTNESS_HIGHEST_ACCEPTED_MAE20_628HC6A8A8A2 = (
    policy
    .sort_values(
        [
            "accepted_mae",
            "selective_gain_mae",
        ],
        ascending=[
            False,
            True,
        ],
    )
    [worst_columns]
    .head(20)
    .reset_index(drop=True)
)

FD001_ROBUSTNESS_LARGEST_COVERAGE_DRIFT20_628HC6A8A8A2 = (
    policy
    .sort_values(
        "abs_coverage_error_vs_nominal",
        ascending=False,
    )
    [
        [
            "scenario_number",
            "scenario_id",
            "family",
            "severity",
            "sensor",
            "coverage",
            "coverage_error_vs_nominal",
            "abs_coverage_error_vs_nominal",
            "selective_gain_mae",
            "accepted_mae",
        ]
    ]
    .head(20)
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------------------
# 10. Contract audit
# Performance results are NOT pass/fail conditions.
# --------------------------------------------------------------------------------------------------
metric_columns = [
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_gain_mae",
    "risk_median",
    "risk_mean",
    "risk_max",
]

metrics_finite = bool(
    np.isfinite(
        policy[
            metric_columns
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)

family_counts = (
    policy["family"]
    .value_counts()
    .to_dict()
)

family_counts_exact = (
    family_counts
    ==
    {
        "bias": 90,
        "drift": 90,
        "noise": 45,
        "dropout": 45,
        "stuck": 45,
    }
)

threshold_after = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

audit_rows = [
    (
        "previous_A8_A1_passed",
        bool(
            C6_A8_A8_A1_AUDIT[
                "passed"
            ].all()
        ),
    ),
    (
        "policy_rows_315",
        policy.shape[0] == 315,
    ),
    (
        "scenario_ids_unique_315",
        policy["scenario_id"].nunique() == 315,
    ),
    (
        "family_counts_exact",
        family_counts_exact,
    ),
    (
        "risk_matrix_shape_315x3272",
        risk_matrix.shape == (315, 3272),
    ),
    (
        "risk_matrix_finite",
        bool(
            np.isfinite(
                risk_matrix
            ).all()
        ),
    ),
    (
        "policy_metrics_finite",
        metrics_finite,
    ),
    (
        "coverage_in_0_1",
        bool(
            (
                (policy["coverage"] >= 0.0)
                &
                (policy["coverage"] <= 1.0)
            ).all()
        ),
    ),
    (
        "accepted_plus_abstained_3272",
        bool(
            (
                policy["accepted"]
                +
                policy["abstained"]
                ==
                3272
            ).all()
        ),
    ),
    (
        "threshold_unchanged",
        threshold_before
        ==
        threshold_after,
    ),
    (
        "test_features_not_read",
        True,
    ),
    (
        "official_test_labels_not_read",
        True,
    ),
    (
        "test_not_scored",
        True,
    ),
    (
        "no_model_execution",
        True,
    ),
    (
        "no_mc_dropout_execution",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning_or_modification",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

C6_A8_A8_A2_AUDIT = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)

all_checks_passed = bool(
    C6_A8_A8_A2_AUDIT[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------------
# 11. Output
# --------------------------------------------------------------------------------------------------
print(
    "\n--- GLOBAL ROBUSTNESS SUMMARY ---"
)

display(
    FD001_ROBUSTNESS_GLOBAL_628HC6A8A8A2
)

print(
    "\n--- FAMILY × SEVERITY ROBUSTNESS ---"
)

display(
    FD001_ROBUSTNESS_FAMILY_SEVERITY_628HC6A8A8A2
)

print(
    "\n--- FAMILY-LEVEL ROBUSTNESS ---"
)

display(
    FD001_ROBUSTNESS_FAMILY_628HC6A8A8A2
)

print(
    "\n--- SENSOR VULNERABILITY RANKING ---"
)

display(
    FD001_ROBUSTNESS_SENSOR_628HC6A8A8A2
)

print(
    "\n--- 20 LOWEST SELECTIVE-GAIN SCENARIOS ---"
)

display(
    FD001_ROBUSTNESS_LOWEST_GAIN20_628HC6A8A8A2
)

print(
    "\n--- 20 HIGHEST ACCEPTED-MAE SCENARIOS ---"
)

display(
    FD001_ROBUSTNESS_HIGHEST_ACCEPTED_MAE20_628HC6A8A8A2
)

print(
    "\n--- 20 LARGEST COVERAGE-DEVIATION SCENARIOS ---"
)

display(
    FD001_ROBUSTNESS_LARGEST_COVERAGE_DRIFT20_628HC6A8A8A2
)

print(
    "\n--- C6-A8-A8-A2 AUDIT ---"
)

display(
    C6_A8_A8_A2_AUDIT
)

print("\n" + "=" * 122)
print("STEP 6.28H-C6-A8-A8-A2 FINAL SUMMARY")
print("=" * 122)

print(
    "Frozen threshold unchanged                :",
    threshold_after,
)

print(
    "Clean selective gain (MAE)                :",
    clean_gain,
)

print(
    "Median corrupted selective gain (MAE)     :",
    float(
        policy[
            "selective_gain_mae"
        ].median()
    ),
)

print(
    "Minimum corrupted selective gain (MAE)    :",
    float(
        policy[
            "selective_gain_mae"
        ].min()
    ),
)

print(
    "Positive-gain scenarios                   :",
    positive_gain_count,
    "/315",
)

print(
    "Accepted-error < abstained-error scenarios:",
    positive_separation_count,
    "/315",
)

print(
    "Coverage within ±0.01 of nominal          :",
    within_1pct_count,
    "/315",
)

print(
    "Coverage within ±0.02 of nominal          :",
    within_2pct_count,
    "/315",
)

print(
    "Median gain retention vs CLEAN            :",
    float(
        policy[
            "gain_retention_ratio_vs_clean"
        ].median()
    ),
)

print(
    "FD001 TEST features read                  : False"
)

print(
    "Official FD001 TEST labels read           : False"
)

print(
    "TEST scored                               : False"
)

print(
    "Model executed in this step               : False"
)

print(
    "MC dropout executed in this step          : False"
)

print(
    "Model retrained                           : False"
)

print(
    "Threshold tuned/modified                  : False"
)

print(
    "Files written                             : False"
)

print(
    "All C6-A8-A8-A2 contract checks passed    :",
    all_checks_passed,
)

if not all_checks_passed:

    failed = C6_A8_A8_A2_AUDIT.loc[
        ~C6_A8_A8_A2_AUDIT[
            "passed"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "C6-A8-A8-A2 contract audit failed: "
        + ", ".join(failed)
    )

print(
    "\nPASS — frozen-policy robustness diagnostics completed "
    "without changing the policy or accessing TEST."
)

print(
    "This step is diagnostic only. "
    "No threshold, model, or reliability rule was selected or modified."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A8-A2 ==="
)

In [ ]:
# ================================================================================================
# STEP 6.28H-C6-A8-A9 — FD001 PRE-TEST FROZEN-POLICY LOCK
# No TEST access. No model/MC execution. No retraining. No threshold modification. No files.
# ================================================================================================

import hashlib
import numpy as np
import pandas as pd

print("=" * 116)
print("STEP 6.28H-C6-A8-A9 — FD001 PRE-TEST FROZEN-POLICY LOCK")
print("=" * 116)

required = [
    "C6_A8_A8_A2_AUDIT",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "FD001_PHASE_EDGES_628HC6A7A2A2",
    "FD001_PHASE_REFERENCES_628HC6A7A2A2",
    "FD001_G1_REFERENCE_628HC6A7A2A2",
    "FD001_G2_REFERENCE_VALUES_628HC6A7A2A2",
    "FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2",
    "fd001_phase_adjust_628hc6a7a2a2",
    "fd001_g1_risk_628hc6a7a2a2",
    "fd001_final_risk_628hc6a7a2a2",
    "FD001_CORRUPTED_POLICY_ROBUSTNESS_628HC6A8A8A2",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing)
    )

if not bool(C6_A8_A8_A2_AUDIT["passed"].all()):
    raise RuntimeError(
        "C6-A8-A8-A2 must pass before the pre-TEST lock."
    )

# --------------------------------------------------------------------------------------------------
# 1. Copy the exact frozen calibration state
# --------------------------------------------------------------------------------------------------

threshold = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

phase_edges = np.asarray(
    FD001_PHASE_EDGES_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1).copy()

phase_refs = {
    int(k):
        np.asarray(
            v,
            dtype=np.float64,
        ).reshape(-1).copy()
    for k, v in FD001_PHASE_REFERENCES_628HC6A7A2A2.items()
}

g1_ref = np.asarray(
    FD001_G1_REFERENCE_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1).copy()

g2_values = np.asarray(
    FD001_G2_REFERENCE_VALUES_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1).copy()

g2_weights = np.asarray(
    FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2,
    dtype=np.float64,
).reshape(-1).copy()

policy = (
    FD001_CORRUPTED_POLICY_ROBUSTNESS_628HC6A8A8A2
    .copy(deep=True)
)

# --------------------------------------------------------------------------------------------------
# 2. Deterministic fingerprints
# --------------------------------------------------------------------------------------------------

def sha256_array_628hc6a8a9(x):

    a = np.ascontiguousarray(
        np.asarray(
            x,
            dtype=np.float64,
        ).reshape(-1)
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def sha256_phase_refs_628hc6a8a9(refs):

    h = hashlib.sha256()

    for key in sorted(refs):

        a = np.ascontiguousarray(
            np.asarray(
                refs[key],
                dtype=np.float64,
            ).reshape(-1)
        )

        h.update(
            np.asarray(
                [
                    key,
                    a.size,
                ],
                dtype=np.int64,
            ).tobytes()
        )

        h.update(
            a.tobytes(order="C")
        )

    return h.hexdigest()


fingerprints = {

    "phase_edges":
        sha256_array_628hc6a8a9(
            phase_edges
        ),

    "phase_references":
        sha256_phase_refs_628hc6a8a9(
            phase_refs
        ),

    "g1_reference":
        sha256_array_628hc6a8a9(
            g1_ref
        ),

    "g2_reference_values":
        sha256_array_628hc6a8a9(
            g2_values
        ),

    "g2_reference_weights":
        sha256_array_628hc6a8a9(
            g2_weights
        ),
}

# --------------------------------------------------------------------------------------------------
# 3. Structural validation
# --------------------------------------------------------------------------------------------------

phase_refs_valid = (
    sorted(
        phase_refs.keys()
    )
    ==
    [
        0,
        1,
        2,
        3,
        4,
    ]
    and
    all(
        ref.size > 0
        and
        np.isfinite(ref).all()
        for ref in phase_refs.values()
    )
)

audit_rows = [

    (
        "previous_A8_A2_passed",
        bool(
            C6_A8_A8_A2_AUDIT[
                "passed"
            ].all()
        ),
    ),

    (
        "threshold_finite_in_0_1",
        bool(
            np.isfinite(threshold)
            and
            0.0 <= threshold <= 1.0
        ),
    ),

    (
        "phase_edges_shape_4",
        phase_edges.shape == (4,),
    ),

    (
        "phase_edges_finite_increasing",
        bool(
            np.isfinite(
                phase_edges
            ).all()
            and
            np.all(
                np.diff(
                    phase_edges
                )
                >
                0.0
            )
        ),
    ),

    (
        "phase_references_valid",
        phase_refs_valid,
    ),

    (
        "g1_reference_shape_3272",
        g1_ref.shape == (3272,),
    ),

    (
        "g1_reference_finite",
        bool(
            np.isfinite(
                g1_ref
            ).all()
        ),
    ),

    (
        "g2_values_shape_3272",
        g2_values.shape == (3272,),
    ),

    (
        "g2_weights_shape_3272",
        g2_weights.shape == (3272,),
    ),

    (
        "g2_values_finite",
        bool(
            np.isfinite(
                g2_values
            ).all()
        ),
    ),

    (
        "g2_weights_positive_finite",
        bool(
            np.isfinite(
                g2_weights
            ).all()
            and
            np.all(
                g2_weights > 0.0
            )
        ),
    ),

    (
        "g2_weights_sum_1",
        bool(
            np.isclose(
                g2_weights.sum(
                    dtype=np.float64
                ),
                1.0,
                rtol=0.0,
                atol=1e-12,
            )
        ),
    ),

    (
        "phase_adjust_callable",
        callable(
            fd001_phase_adjust_628hc6a7a2a2
        ),
    ),

    (
        "g1_risk_callable",
        callable(
            fd001_g1_risk_628hc6a7a2a2
        ),
    ),

    (
        "final_risk_callable",
        callable(
            fd001_final_risk_628hc6a7a2a2
        ),
    ),

    (
        "robustness_rows_315",
        policy.shape[0] == 315,
    ),

    (
        "scenario_ids_unique_315",
        policy[
            "scenario_id"
        ].nunique()
        ==
        315,
    ),

    (
        "test_features_not_read",
        True,
    ),

    (
        "official_test_labels_not_read",
        True,
    ),

    (
        "test_not_scored",
        True,
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_mc_dropout_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

C6_A8_A9_AUDIT = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)

all_passed = bool(
    C6_A8_A9_AUDIT[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------------
# 4. Record the already-observed DEV robustness state
#    This is descriptive only; it does not tune or select anything.
# --------------------------------------------------------------------------------------------------

diagnostic_snapshot = {

    "scenarios":
        int(
            policy.shape[0]
        ),

    "median_coverage":
        float(
            policy[
                "coverage"
            ].median()
        ),

    "median_selective_gain_mae":
        float(
            policy[
                "selective_gain_mae"
            ].median()
        ),

    "minimum_selective_gain_mae":
        float(
            policy[
                "selective_gain_mae"
            ].min()
        ),

    "positive_gain_scenarios":
        int(
            (
                policy[
                    "selective_gain_mae"
                ]
                >
                0.0
            ).sum()
        ),

    "accepted_better_than_abstained":
        int(
            (
                policy[
                    "abstained_minus_accepted_mae"
                ]
                >
                0.0
            ).sum()
        ),

    "coverage_within_abs_0p01":
        int(
            (
                np.abs(
                    policy[
                        "coverage"
                    ]
                    -
                    0.80
                )
                <=
                0.01
            ).sum()
        ),

    "coverage_within_abs_0p02":
        int(
            (
                np.abs(
                    policy[
                        "coverage"
                    ]
                    -
                    0.80
                )
                <=
                0.02
            ).sum()
        ),
}

# --------------------------------------------------------------------------------------------------
# 5. In-memory frozen lock
# --------------------------------------------------------------------------------------------------

FD001_PRETEST_POLICY_LOCK_628HC6A8A9 = {

    "dataset":
        "FD001",

    "reliability_architecture":
        "uncertainty_only",

    "nominal_coverage":
        0.80,

    "threshold":
        threshold,

    "threshold_comparator":
        "risk <= threshold",

    "phase_edges":
        phase_edges,

    "phase_references":
        phase_refs,

    "g1_reference":
        g1_ref,

    "g2_reference_values":
        g2_values,

    "g2_reference_weights":
        g2_weights,

    "fingerprints":
        fingerprints,

    "diagnostic_snapshot":
        diagnostic_snapshot,

    "policy_status":
        "FROZEN_UNCHANGED",

    "test_accessed":
        False,
}

# --------------------------------------------------------------------------------------------------
# 6. Output
# --------------------------------------------------------------------------------------------------

print(
    "\n--- FROZEN CALIBRATION FINGERPRINTS ---"
)

display(
    pd.DataFrame(
        [
            {
                "object": key,
                "sha256": value,
            }
            for key, value in fingerprints.items()
        ]
    )
)

print(
    "\n--- LOCKED DEV DIAGNOSTIC SNAPSHOT ---"
)

display(
    pd.DataFrame(
        [
            diagnostic_snapshot
        ]
    )
)

print(
    "\n--- C6-A8-A9 AUDIT ---"
)

display(
    C6_A8_A9_AUDIT
)

print(
    "\n"
    +
    "=" * 116
)

print(
    "STEP 6.28H-C6-A8-A9 FINAL SUMMARY"
)

print(
    "=" * 116
)

print(
    "Frozen threshold                 :",
    threshold,
)

print(
    "Threshold comparator              : risk <= threshold"
)

print(
    "Nominal coverage                  : 0.80"
)

print(
    "Policy status                     : FROZEN_UNCHANGED"
)

print(
    "Positive-gain DEV scenarios       :",
    diagnostic_snapshot[
        "positive_gain_scenarios"
    ],
    "/315",
)

print(
    "Accepted-error < abstained-error  :",
    diagnostic_snapshot[
        "accepted_better_than_abstained"
    ],
    "/315",
)

print(
    "Coverage within ±0.01             :",
    diagnostic_snapshot[
        "coverage_within_abs_0p01"
    ],
    "/315",
)

print(
    "Coverage within ±0.02             :",
    diagnostic_snapshot[
        "coverage_within_abs_0p02"
    ],
    "/315",
)

print(
    "FD001 TEST features read          : False"
)

print(
    "Official FD001 TEST labels read   : False"
)

print(
    "TEST scored                       : False"
)

print(
    "Model executed in this step       : False"
)

print(
    "MC dropout executed in this step  : False"
)

print(
    "Model retrained                   : False"
)

print(
    "Threshold tuned/modified          : False"
)

print(
    "Files written                     : False"
)

print(
    "All C6-A8-A9 checks passed        :",
    all_passed,
)

if not all_passed:

    failed = C6_A8_A9_AUDIT.loc[
        ~C6_A8_A9_AUDIT[
            "passed"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "C6-A8-A9 pre-TEST lock failed: "
        +
        ", ".join(
            failed
        )
    )

print(
    "\nPASS — frozen FD001 policy is locked in memory "
    "for the eventual TEST stage."
)

print(
    "No TEST data were accessed and no policy component was changed."
)

print(
    "\n=== END STEP 6.28H-C6-A8-A9 ==="
)

In [ ]:
# ================================================================================================
# STEP 6.28H-C6-A9-A0 — FD001 TEST INPUT / EXECUTION CONTRACT DISCOVERY
#
# First post-freeze TEST-entry step.
#
# PURPOSE
#   1. Verify the frozen FD001 policy lock remains intact.
#   2. Identify exact in-memory FD001 TEST feature / RUL candidates.
#   3. Identify the exact A5 normalization/window bridge.
#   4. Lock the exact FD001 model + MC-dropout helper.
#
# IMPORTANT
#   - metadata inspection only
#   - NO model forward pass
#   - NO MC dropout
#   - NO TEST scoring
#   - NO retraining
#   - NO threshold modification
#   - NO files written
# ================================================================================================

import hashlib
import inspect
import numpy as np
import pandas as pd
import torch


print("=" * 116)
print(
    "STEP 6.28H-C6-A9-A0 — "
    "FD001 TEST INPUT / EXECUTION CONTRACT DISCOVERY"
)
print("=" * 116)


# ------------------------------------------------------------------------------------------------
# 1. REQUIRE THE COMPLETED PRE-TEST LOCK
# ------------------------------------------------------------------------------------------------

if "C6_A8_A9_AUDIT" not in globals():

    raise RuntimeError(
        "C6-A8-A9 audit object is missing."
    )


if not bool(
    C6_A8_A9_AUDIT[
        "passed"
    ].all()
):

    raise RuntimeError(
        "C6-A8-A9 must pass before "
        "any TEST-entry inspection."
    )


if (
    "FD001_PRETEST_POLICY_LOCK_628HC6A8A9"
    not in globals()
):

    raise RuntimeError(
        "FD001 pre-TEST policy lock object "
        "is missing."
    )


policy_lock = (
    FD001_PRETEST_POLICY_LOCK_628HC6A8A9
)


if (
    policy_lock.get(
        "policy_status"
    )
    !=
    "FROZEN_UNCHANGED"
):

    raise RuntimeError(
        "Frozen policy status is not "
        "FROZEN_UNCHANGED."
    )


locked_threshold = float(
    policy_lock[
        "threshold"
    ]
)


locked_fingerprints = dict(
    policy_lock[
        "fingerprints"
    ]
)


# ------------------------------------------------------------------------------------------------
# 2. STABLE GLOBALS SNAPSHOT
# ------------------------------------------------------------------------------------------------

G = list(
    globals().items()
)


# ------------------------------------------------------------------------------------------------
# 3. TEST FEATURE CANDIDATES
#
# Metadata only:
#   name
#   object type
#   shape
#   schema hints
#
# No TEST values are sent to the model.
# ------------------------------------------------------------------------------------------------

feature_rows = []


for name, obj in G:

    lname = name.lower()

    if "test" not in lname:

        continue


    if isinstance(
        obj,
        pd.DataFrame,
    ):

        cols = [
            str(c)
            for c in obj.columns
        ]


        feature_rows.append(
            {
                "name":
                    name,

                "type":
                    "DataFrame",

                "shape":
                    tuple(
                        obj.shape
                    ),

                "fd001_name_hint":
                    (
                        "fd001"
                        in lname
                    ),

                "has_unit_cycle":
                    (
                        (
                            "unit"
                            in cols
                            or
                            "unit_id"
                            in cols
                        )
                        and
                        "cycle"
                        in cols
                    ),

                "sensor_column_count":
                    int(
                        sum(
                            c.startswith(
                                "sensor_"
                            )
                            for c in cols
                        )
                    ),

                "columns_preview":
                    cols[:12],
            }
        )


    elif isinstance(
        obj,
        np.ndarray,
    ):

        feature_rows.append(
            {
                "name":
                    name,

                "type":
                    "ndarray",

                "shape":
                    tuple(
                        obj.shape
                    ),

                "fd001_name_hint":
                    (
                        "fd001"
                        in lname
                    ),

                "has_unit_cycle":
                    False,

                "sensor_column_count":
                    0,

                "columns_preview":
                    None,
            }
        )


test_feature_candidates = pd.DataFrame(
    feature_rows
)


if not test_feature_candidates.empty:

    test_feature_candidates = (
        test_feature_candidates
        .sort_values(
            [
                "fd001_name_hint",
                "type",
                "name",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------------------------------------
# 4. OFFICIAL TEST-LABEL / RUL CANDIDATES
#
# Metadata only.
# Values are NOT displayed and NOT scored.
# ------------------------------------------------------------------------------------------------

label_tokens = (
    "rul",
    "label",
    "target",
    "y_test",
)


label_rows = []


for name, obj in G:

    lname = name.lower()


    if "test" not in lname:

        continue


    if not any(
        token in lname
        for token in label_tokens
    ):

        continue


    if isinstance(
        obj,
        pd.DataFrame,
    ):

        shape = tuple(
            obj.shape
        )

        obj_type = (
            "DataFrame"
        )


    elif isinstance(
        obj,
        pd.Series,
    ):

        shape = tuple(
            obj.shape
        )

        obj_type = (
            "Series"
        )


    elif isinstance(
        obj,
        np.ndarray,
    ):

        shape = tuple(
            obj.shape
        )

        obj_type = (
            "ndarray"
        )


    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):

        shape = (
            len(obj),
        )

        obj_type = (
            type(obj).__name__
        )


    else:

        continue


    label_rows.append(
        {
            "name":
                name,

            "type":
                obj_type,

            "shape":
                shape,

            "fd001_name_hint":
                (
                    "fd001"
                    in lname
                ),
        }
    )


test_label_candidates = pd.DataFrame(
    label_rows
)


if not test_label_candidates.empty:

    test_label_candidates = (
        test_label_candidates
        .sort_values(
            [
                "fd001_name_hint",
                "name",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------------------------------------
# 5. EXACT A5 NORMALIZATION / WINDOW-BRIDGE CANDIDATES
# ------------------------------------------------------------------------------------------------

bridge_rows = []


for name, obj in G:

    if not callable(
        obj
    ):

        continue


    lname = name.lower()


    if (
        "628hc6a8a5"
        not in lname
        and
        not (
            "fd001"
            in lname
            and
            (
                "normal"
                in lname
                or
                "window"
                in lname
                or
                "metadata"
                in lname
            )
        )
    ):

        continue


    try:

        signature = str(
            inspect.signature(
                obj
            )
        )


    except Exception:

        signature = (
            "<signature unavailable>"
        )


    bridge_rows.append(
        {
            "name":
                name,

            "signature":
                signature,
        }
    )


bridge_candidates = pd.DataFrame(
    bridge_rows
)


if not bridge_candidates.empty:

    bridge_candidates = (
        bridge_candidates
        .sort_values(
            "name"
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------------------------------------
# 6. LOCK THE ALREADY-IDENTIFIED MODEL + MC HELPER
#
# No forward pass is performed.
# ------------------------------------------------------------------------------------------------

model_name = (
    "MODEL_FD001_628HC6A6A1"
)


mc_helper_name = (
    "mc_dropout_predict_628hc0"
)


model_present = (
    model_name
    in globals()
)


mc_helper_present = (
    mc_helper_name
    in globals()
    and
    callable(
        globals()[
            mc_helper_name
        ]
    )
)


model_parameter_count = None
model_training_mode = None
model_state_sha256 = None


if model_present:

    model = globals()[
        model_name
    ]


    model_parameter_count = int(
        sum(
            p.numel()
            for p
            in model.parameters()
        )
    )


    model_training_mode = bool(
        model.training
    )


    h = hashlib.sha256()


    for key, value in sorted(
        model.state_dict().items()
    ):

        h.update(
            key.encode(
                "utf-8"
            )
        )


        a = (
            value
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )


        h.update(
            str(
                a.dtype
            ).encode(
                "utf-8"
            )
        )


        h.update(
            np.asarray(
                a.shape,
                dtype=np.int64,
            ).tobytes()
        )


        h.update(
            a.tobytes(
                order="C"
            )
        )


    model_state_sha256 = (
        h.hexdigest()
    )


mc_helper_signature = None


if mc_helper_present:

    mc_helper_signature = str(
        inspect.signature(
            globals()[
                mc_helper_name
            ]
        )
    )


# ------------------------------------------------------------------------------------------------
# 7. CONTRACT AUDIT
# ------------------------------------------------------------------------------------------------

audit_rows = [

    (
        "pretest_policy_lock_passed",
        True,
    ),

    (
        "policy_status_frozen_unchanged",
        (
            policy_lock.get(
                "policy_status"
            )
            ==
            "FROZEN_UNCHANGED"
        ),
    ),

    (
        "locked_threshold_finite",
        bool(
            np.isfinite(
                locked_threshold
            )
        ),
    ),

    (
        "locked_fingerprint_count_5",
        (
            len(
                locked_fingerprints
            )
            ==
            5
        ),
    ),

    (
        "fd001_model_present",
        model_present,
    ),

    (
        "fd001_model_parameter_count_18049",
        (
            model_parameter_count
            ==
            18049
        ),
    ),

    (
        "fd001_model_in_eval_mode",
        (
            model_training_mode
            is False
        ),
    ),

    (
        "mc_dropout_helper_present",
        mc_helper_present,
    ),

    (
        "no_model_forward_pass",
        True,
    ),

    (
        "no_mc_dropout_execution",
        True,
    ),

    (
        "no_test_scoring",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]


C6_A9_A0_AUDIT = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)


all_passed = bool(
    C6_A9_A0_AUDIT[
        "passed"
    ].all()
)


# ------------------------------------------------------------------------------------------------
# 8. OUTPUT
# ------------------------------------------------------------------------------------------------

print(
    "\n--- TEST FEATURE CANDIDATES: "
    "METADATA ONLY ---"
)


if test_feature_candidates.empty:

    print(
        "No in-memory TEST feature "
        "candidates found."
    )


else:

    display(
        test_feature_candidates
    )


print(
    "\n--- TEST LABEL / RUL CANDIDATES: "
    "METADATA ONLY ---"
)


if test_label_candidates.empty:

    print(
        "No in-memory TEST label/RUL "
        "candidates found."
    )


else:

    display(
        test_label_candidates
    )


print(
    "\n--- A5 NORMALIZATION / "
    "WINDOW BRIDGE CANDIDATES ---"
)


if bridge_candidates.empty:

    print(
        "No callable A5 bridge "
        "candidates found."
    )


else:

    display(
        bridge_candidates
    )


print(
    "\n--- EXECUTION OBJECT LOCK ---"
)


print(
    "Model object              :",
    (
        model_name
        if model_present
        else None
    ),
)


print(
    "Model parameter count     :",
    model_parameter_count,
)


print(
    "Model training mode       :",
    model_training_mode,
)


print(
    "Model state SHA-256       :",
    model_state_sha256,
)


print(
    "MC helper                 :",
    (
        mc_helper_name
        if mc_helper_present
        else None
    ),
)


print(
    "MC helper signature       :",
    mc_helper_signature,
)


print(
    "Frozen threshold          :",
    locked_threshold,
)


print(
    "Threshold comparator      : "
    "risk <= threshold"
)


print(
    "\n--- C6-A9-A0 AUDIT ---"
)


display(
    C6_A9_A0_AUDIT
)


print(
    "\n"
    +
    "=" * 116
)


print(
    "STEP 6.28H-C6-A9-A0 "
    "FINAL SUMMARY"
)


print(
    "=" * 116
)


print(
    "Pre-TEST policy remains frozen       :",
    policy_lock.get(
        "policy_status"
    ),
)


print(
    "TEST candidate metadata inspected    : True"
)


print(
    "TEST feature values sent to model    : False"
)


print(
    "Official TEST labels used for score  : False"
)


print(
    "TEST scored                           : False"
)


print(
    "Model forward pass executed           : False"
)


print(
    "MC dropout executed                   : False"
)


print(
    "Model retrained                       : False"
)


print(
    "Threshold tuned/modified              : False"
)


print(
    "Files written                         : False"
)


print(
    "All C6-A9-A0 contract checks passed   :",
    all_passed,
)


if not all_passed:

    failed = (
        C6_A9_A0_AUDIT
        .loc[
            ~C6_A9_A0_AUDIT[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )


    raise RuntimeError(
        "C6-A9-A0 execution-contract "
        "discovery failed: "
        +
        ", ".join(
            failed
        )
    )


print(
    "\nPASS — the frozen policy and execution "
    "objects remain intact."
)


print(
    "Use the candidate tables above to build "
    "the exact one-way TEST pipeline."
)


print(
    "\n=== END STEP 6.28H-C6-A9-A0 ==="
)

In [ ]:
# ================================================================================================
# STEP 6.28H-C6-A9-A1 — EXACT FD001 TEST / RUL SOURCE-PATH LOCK
#
# PURPOSE
#   Resolve the exact official FD001 TEST feature source and official FD001 RUL source
#   BEFORE reading either file.
#
# THIS CELL DOES NOT:
#   - open TEST files
#   - read TEST feature values
#   - read official TEST labels
#   - run the model
#   - run MC dropout
#   - score TEST
#   - change the frozen threshold
#   - write files
# ================================================================================================

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 116)
print(
    "STEP 6.28H-C6-A9-A1 — "
    "EXACT FD001 TEST / RUL SOURCE-PATH LOCK"
)
print("=" * 116)


# ------------------------------------------------------------------------------------------------
# 1. REQUIRE PREVIOUS LOCK
# ------------------------------------------------------------------------------------------------

if "C6_A9_A0_AUDIT" not in globals():

    raise RuntimeError(
        "C6-A9-A0 audit object is missing."
    )


if not bool(
    C6_A9_A0_AUDIT["passed"].all()
):

    raise RuntimeError(
        "C6-A9-A0 must pass first."
    )


if (
    "FD001_PRETEST_POLICY_LOCK_628HC6A8A9"
    not in globals()
):

    raise RuntimeError(
        "Frozen FD001 pre-TEST policy lock is missing."
    )


policy_lock_628hc6a9a1 = (
    FD001_PRETEST_POLICY_LOCK_628HC6A8A9
)


if (
    policy_lock_628hc6a9a1.get(
        "policy_status"
    )
    !=
    "FROZEN_UNCHANGED"
):

    raise RuntimeError(
        "FD001 policy is not frozen."
    )


# ------------------------------------------------------------------------------------------------
# 2. STABLE GLOBAL SNAPSHOT
# ------------------------------------------------------------------------------------------------

G_628hc6a9a1 = list(
    globals().items()
)


# ------------------------------------------------------------------------------------------------
# 3. SEARCH PATH-LIKE OBJECTS ONLY
#
# We inspect strings / Path objects / string lists.
# No referenced file is opened.
# ------------------------------------------------------------------------------------------------

path_rows_628hc6a9a1 = []


def add_path_candidate_628hc6a9a1(
    source_name,
    value,
    source_type,
):

    text = str(
        value
    )

    lower = (
        text.lower()
        .replace("\\", "/")
    )


    has_fd001 = (
        "fd001"
        in lower
    )


    has_test = (
        "test"
        in lower
    )


    has_rul = (
        "rul"
        in lower
    )


    looks_txt = (
        lower.endswith(
            ".txt"
        )
    )


    if not (
        has_fd001
        or
        (
            has_test
            and
            has_rul
        )
    ):

        return


    path_rows_628hc6a9a1.append(
        {
            "source_object":
                source_name,

            "source_type":
                source_type,

            "value":
                text,

            "has_fd001":
                has_fd001,

            "has_test":
                has_test,

            "has_rul":
                has_rul,

            "looks_txt":
                looks_txt,

            "candidate_role":
                (
                    "FD001_RUL"
                    if (
                        has_fd001
                        and
                        has_rul
                    )
                    else
                    "FD001_TEST"
                    if (
                        has_fd001
                        and
                        has_test
                    )
                    else
                    "OTHER"
                ),
        }
    )


for name, obj in G_628hc6a9a1:

    if isinstance(
        obj,
        (
            str,
            Path,
        ),
    ):

        add_path_candidate_628hc6a9a1(
            name,
            obj,
            type(obj).__name__,
        )


    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):

        for idx, item in enumerate(
            obj
        ):

            if isinstance(
                item,
                (
                    str,
                    Path,
                ),
            ):

                add_path_candidate_628hc6a9a1(
                    f"{name}[{idx}]",
                    item,
                    type(obj).__name__,
                )


path_candidates_628hc6a9a1 = pd.DataFrame(
    path_rows_628hc6a9a1
)


if not path_candidates_628hc6a9a1.empty:

    path_candidates_628hc6a9a1 = (
        path_candidates_628hc6a9a1
        .drop_duplicates(
            subset=[
                "value",
                "candidate_role",
            ]
        )
        .sort_values(
            [
                "candidate_role",
                "value",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------------------------------------
# 4. SEARCH ALREADY-LOADED FD001 TEST OBJECTS BY NAME ONLY
#
# No values inspected.
# ------------------------------------------------------------------------------------------------

loaded_rows_628hc6a9a1 = []


for name, obj in G_628hc6a9a1:

    lname = (
        name.lower()
    )


    if not (
        "fd001"
        in lname
        and
        "test"
        in lname
    ):

        continue


    if isinstance(
        obj,
        pd.DataFrame,
    ):

        loaded_rows_628hc6a9a1.append(
            {
                "name":
                    name,

                "type":
                    "DataFrame",

                "shape":
                    tuple(
                        obj.shape
                    ),

                "columns":
                    list(
                        map(
                            str,
                            obj.columns,
                        )
                    ),
            }
        )


    elif isinstance(
        obj,
        pd.Series,
    ):

        loaded_rows_628hc6a9a1.append(
            {
                "name":
                    name,

                "type":
                    "Series",

                "shape":
                    tuple(
                        obj.shape
                    ),

                "columns":
                    None,
            }
        )


    elif isinstance(
        obj,
        np.ndarray,
    ):

        loaded_rows_628hc6a9a1.append(
            {
                "name":
                    name,

                "type":
                    "ndarray",

                "shape":
                    tuple(
                        obj.shape
                    ),

                "columns":
                    None,
            }
        )


loaded_fd001_test_objects_628hc6a9a1 = pd.DataFrame(
    loaded_rows_628hc6a9a1
)


if not loaded_fd001_test_objects_628hc6a9a1.empty:

    loaded_fd001_test_objects_628hc6a9a1 = (
        loaded_fd001_test_objects_628hc6a9a1
        .sort_values(
            "name"
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------------------------------------
# 5. EXTRACT UNAMBIGUOUS PATH CANDIDATES
#
# Still no file opening.
# ------------------------------------------------------------------------------------------------

fd001_test_path_rows_628hc6a9a1 = []

fd001_rul_path_rows_628hc6a9a1 = []


if not path_candidates_628hc6a9a1.empty:

    fd001_test_path_rows_628hc6a9a1 = (
        path_candidates_628hc6a9a1
        .loc[
            path_candidates_628hc6a9a1[
                "candidate_role"
            ]
            ==
            "FD001_TEST"
        ]
        .copy()
    )


    fd001_rul_path_rows_628hc6a9a1 = (
        path_candidates_628hc6a9a1
        .loc[
            path_candidates_628hc6a9a1[
                "candidate_role"
            ]
            ==
            "FD001_RUL"
        ]
        .copy()
    )


test_path_count_628hc6a9a1 = (
    len(
        fd001_test_path_rows_628hc6a9a1
    )
    if isinstance(
        fd001_test_path_rows_628hc6a9a1,
        pd.DataFrame,
    )
    else 0
)


rul_path_count_628hc6a9a1 = (
    len(
        fd001_rul_path_rows_628hc6a9a1
    )
    if isinstance(
        fd001_rul_path_rows_628hc6a9a1,
        pd.DataFrame,
    )
    else 0
)


# ------------------------------------------------------------------------------------------------
# 6. AUDIT
# ------------------------------------------------------------------------------------------------

audit_rows_628hc6a9a1 = [

    (
        "previous_A9_A0_passed",
        True,
    ),

    (
        "policy_status_frozen_unchanged",
        (
            policy_lock_628hc6a9a1.get(
                "policy_status"
            )
            ==
            "FROZEN_UNCHANGED"
        ),
    ),

    (
        "frozen_threshold_unchanged",
        bool(
            np.isclose(
                float(
                    policy_lock_628hc6a9a1[
                        "threshold"
                    ]
                ),
                0.7983391809203397,
                rtol=0.0,
                atol=0.0,
            )
        ),
    ),

    (
        "no_test_file_opened",
        True,
    ),

    (
        "no_test_feature_values_read",
        True,
    ),

    (
        "official_test_labels_not_read",
        True,
    ),

    (
        "test_not_scored",
        True,
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_mc_dropout_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]


C6_A9_A1_AUDIT = pd.DataFrame(
    audit_rows_628hc6a9a1,
    columns=[
        "check",
        "passed",
    ],
)


all_passed_628hc6a9a1 = bool(
    C6_A9_A1_AUDIT[
        "passed"
    ].all()
)


# ------------------------------------------------------------------------------------------------
# 7. OUTPUT
# ------------------------------------------------------------------------------------------------

print(
    "\n--- FD001 PATH CANDIDATES ---"
)


if path_candidates_628hc6a9a1.empty:

    print(
        "No FD001 path-like candidates were found."
    )


else:

    display(
        path_candidates_628hc6a9a1
    )


print(
    "\n--- ALREADY-LOADED FD001 TEST OBJECTS ---"
)


if loaded_fd001_test_objects_628hc6a9a1.empty:

    print(
        "No explicitly named FD001 TEST object is currently loaded."
    )


else:

    display(
        loaded_fd001_test_objects_628hc6a9a1
    )


print(
    "\n--- EXACT ROLE COUNTS ---"
)


print(
    "FD001 TEST path candidate count :",
    test_path_count_628hc6a9a1,
)


print(
    "FD001 RUL path candidate count  :",
    rul_path_count_628hc6a9a1,
)


print(
    "\n--- C6-A9-A1 AUDIT ---"
)


display(
    C6_A9_A1_AUDIT
)


print(
    "\n"
    +
    "=" * 116
)


print(
    "STEP 6.28H-C6-A9-A1 FINAL SUMMARY"
)


print(
    "=" * 116
)


print(
    "Frozen policy status              :",
    policy_lock_628hc6a9a1.get(
        "policy_status"
    ),
)


print(
    "Frozen threshold                  :",
    float(
        policy_lock_628hc6a9a1[
            "threshold"
        ]
    ),
)


print(
    "FD001 TEST path candidates        :",
    test_path_count_628hc6a9a1,
)


print(
    "FD001 RUL path candidates         :",
    rul_path_count_628hc6a9a1,
)


print(
    "TEST file opened                  : False"
)


print(
    "TEST feature values read          : False"
)


print(
    "Official FD001 TEST labels read   : False"
)


print(
    "TEST scored                       : False"
)


print(
    "Model executed                    : False"
)


print(
    "MC dropout executed               : False"
)


print(
    "Model retrained                   : False"
)


print(
    "Threshold tuned/modified          : False"
)


print(
    "Files written                     : False"
)


print(
    "All C6-A9-A1 checks passed        :",
    all_passed_628hc6a9a1,
)


if not all_passed_628hc6a9a1:

    failed_628hc6a9a1 = (
        C6_A9_A1_AUDIT
        .loc[
            ~C6_A9_A1_AUDIT[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )


    raise RuntimeError(
        "C6-A9-A1 failed: "
        +
        ", ".join(
            failed_628hc6a9a1
        )
    )


print(
    "\nPASS — only TEST/RUL source locations were inspected."
)


print(
    "Do NOT open or score TEST yet. "
    "Send this output back for the exact TEST-read contract."
)


print(
    "\n=== END STEP 6.28H-C6-A9-A1 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A2 — FD001 OFFICIAL TEST SOURCE RESOLUTION + ONE-TIME READ CONTRACT
#
# Reads the official FD001 TEST features and official FD001 endpoint-RUL file exactly once.
# Does NOT normalize, window, run the model, run MC dropout, score TEST, or modify the policy.
# ============================================================================================

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 116)
print(
    "STEP 6.28H-C6-A9-A2 — "
    "FD001 OFFICIAL TEST SOURCE RESOLUTION + ONE-TIME READ CONTRACT"
)
print("=" * 116)


# --------------------------------------------------------------------------------------------
# 1. Require frozen pre-TEST policy
# --------------------------------------------------------------------------------------------

if (
    "C6_A9_A0_AUDIT" not in globals()
    or
    not bool(
        C6_A9_A0_AUDIT["passed"].all()
    )
):
    raise RuntimeError(
        "C6-A9-A0 frozen execution-contract lock must pass first."
    )


if "FD001_PRETEST_POLICY_LOCK_628HC6A8A9" not in globals():
    raise RuntimeError(
        "FD001_PRETEST_POLICY_LOCK_628HC6A8A9 is missing."
    )


policy_lock_628hc6a9a2 = (
    FD001_PRETEST_POLICY_LOCK_628HC6A8A9
)


if (
    policy_lock_628hc6a9a2.get(
        "policy_status"
    )
    !=
    "FROZEN_UNCHANGED"
):
    raise RuntimeError(
        "FD001 policy is not FROZEN_UNCHANGED."
    )


frozen_threshold_before_628hc6a9a2 = float(
    policy_lock_628hc6a9a2[
        "threshold"
    ]
)


# --------------------------------------------------------------------------------------------
# 2. Resolve exact official filenames from filesystem
# --------------------------------------------------------------------------------------------

project_root_628hc6a9a2 = (
    Path.cwd().resolve()
)


test_matches_628hc6a9a2 = sorted(
    {
        p.resolve()
        for p in project_root_628hc6a9a2.rglob(
            "test_FD001.txt"
        )
        if p.is_file()
    }
)


rul_matches_628hc6a9a2 = sorted(
    {
        p.resolve()
        for p in project_root_628hc6a9a2.rglob(
            "RUL_FD001.txt"
        )
        if p.is_file()
    }
)


print(
    "\n--- EXACT FILESYSTEM MATCHES ---"
)

print(
    "Project root :",
    project_root_628hc6a9a2,
)

print(
    "TEST matches :",
    len(
        test_matches_628hc6a9a2
    ),
)

for p in test_matches_628hc6a9a2:
    print(
        "   ",
        p,
    )


print(
    "RUL matches  :",
    len(
        rul_matches_628hc6a9a2
    ),
)

for p in rul_matches_628hc6a9a2:
    print(
        "   ",
        p,
    )


if len(
    test_matches_628hc6a9a2
) != 1:
    raise RuntimeError(
        "Expected exactly one test_FD001.txt under "
        f"{project_root_628hc6a9a2}; "
        f"found {len(test_matches_628hc6a9a2)}."
    )


if len(
    rul_matches_628hc6a9a2
) != 1:
    raise RuntimeError(
        "Expected exactly one RUL_FD001.txt under "
        f"{project_root_628hc6a9a2}; "
        f"found {len(rul_matches_628hc6a9a2)}."
    )


FD001_TEST_PATH_628HC6A9A2 = (
    test_matches_628hc6a9a2[0]
)

FD001_RUL_PATH_628HC6A9A2 = (
    rul_matches_628hc6a9a2[0]
)


# --------------------------------------------------------------------------------------------
# 3. Recover exact raw schema from locked FD001 DEV object
# --------------------------------------------------------------------------------------------

if "FD001_DEV_RAW_628HC6A3R1" not in globals():
    raise RuntimeError(
        "FD001_DEV_RAW_628HC6A3R1 is missing."
    )


dev_raw_contract_628hc6a9a2 = (
    FD001_DEV_RAW_628HC6A3R1
)


if not isinstance(
    dev_raw_contract_628hc6a9a2,
    pd.DataFrame,
):
    raise RuntimeError(
        "FD001_DEV_RAW_628HC6A3R1 is not a DataFrame."
    )


raw_columns_628hc6a9a2 = list(
    dev_raw_contract_628hc6a9a2.columns
)


if len(
    raw_columns_628hc6a9a2
) != 26:
    raise RuntimeError(
        "Expected locked FD001 raw schema to contain "
        f"26 columns; found {len(raw_columns_628hc6a9a2)}."
    )


unit_col_628hc6a9a2 = (
    raw_columns_628hc6a9a2[0]
)

cycle_col_628hc6a9a2 = (
    raw_columns_628hc6a9a2[1]
)


if (
    unit_col_628hc6a9a2
    not in
    {
        "unit",
        "unit_id",
    }
    or
    cycle_col_628hc6a9a2
    !=
    "cycle"
):
    raise RuntimeError(
        "Locked FD001 raw schema has unexpected "
        "identity-column contract: "
        f"{raw_columns_628hc6a9a2[:2]}"
    )


# --------------------------------------------------------------------------------------------
# 4. FIRST intentional TEST read
# --------------------------------------------------------------------------------------------

FD001_TEST_RAW_628HC6A9A2 = pd.read_csv(
    FD001_TEST_PATH_628HC6A9A2,
    sep=r"\s+",
    header=None,
    names=raw_columns_628hc6a9a2,
    engine="python",
)


FD001_TEST_RUL_ENDPOINT_628HC6A9A2 = (
    pd.read_csv(
        FD001_RUL_PATH_628HC6A9A2,
        sep=r"\s+",
        header=None,
        engine="python",
    )
    .iloc[:, 0]
    .astype(
        np.float64
    )
    .to_numpy()
)


# --------------------------------------------------------------------------------------------
# 5. Read-contract validation
# --------------------------------------------------------------------------------------------

test_raw_628hc6a9a2 = (
    FD001_TEST_RAW_628HC6A9A2
)

test_rul_628hc6a9a2 = (
    FD001_TEST_RUL_ENDPOINT_628HC6A9A2
)


test_units_628hc6a9a2 = np.sort(
    test_raw_628hc6a9a2[
        unit_col_628hc6a9a2
    ]
    .astype(int)
    .unique()
)


test_engine_count_628hc6a9a2 = int(
    len(
        test_units_628hc6a9a2
    )
)


rows_per_engine_628hc6a9a2 = (
    test_raw_628hc6a9a2
    .groupby(
        unit_col_628hc6a9a2,
        sort=True,
    )
    .size()
    .astype(int)
)


last_cycle_by_engine_628hc6a9a2 = (
    test_raw_628hc6a9a2
    .groupby(
        unit_col_628hc6a9a2,
        sort=True,
    )[
        cycle_col_628hc6a9a2
    ]
    .max()
    .astype(int)
)


retained_sensors_628hc6a9a2 = list(
    FD001_RETAINED_SENSORS_628HC6A4
)


schema_exact_628hc6a9a2 = (
    list(
        test_raw_628hc6a9a2.columns
    )
    ==
    raw_columns_628hc6a9a2
)


all_numeric_finite_628hc6a9a2 = bool(
    np.isfinite(
        test_raw_628hc6a9a2
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


rul_finite_628hc6a9a2 = bool(
    np.isfinite(
        test_rul_628hc6a9a2
    ).all()
)


rul_nonnegative_628hc6a9a2 = bool(
    (
        test_rul_628hc6a9a2
        >=
        0.0
    ).all()
)


engine_rul_count_match_628hc6a9a2 = (
    test_engine_count_628hc6a9a2
    ==
    len(
        test_rul_628hc6a9a2
    )
)


retained_sensors_present_628hc6a9a2 = (
    set(
        retained_sensors_628hc6a9a2
    )
    .issubset(
        test_raw_628hc6a9a2.columns
    )
)


cycles_positive_628hc6a9a2 = bool(
    (
        test_raw_628hc6a9a2[
            cycle_col_628hc6a9a2
        ]
        .to_numpy(
            dtype=np.float64
        )
        >=
        1.0
    ).all()
)


minimum_engine_history_628hc6a9a2 = int(
    rows_per_engine_628hc6a9a2.min()
)


frozen_threshold_after_628hc6a9a2 = float(
    FD001_PRETEST_POLICY_LOCK_628HC6A8A9[
        "threshold"
    ]
)


threshold_unchanged_628hc6a9a2 = bool(
    frozen_threshold_before_628hc6a9a2
    ==
    frozen_threshold_after_628hc6a9a2
)


# --------------------------------------------------------------------------------------------
# 6. Audit
# --------------------------------------------------------------------------------------------

audit_rows_628hc6a9a2 = [

    (
        "pretest_policy_frozen_unchanged",
        (
            policy_lock_628hc6a9a2.get(
                "policy_status"
            )
            ==
            "FROZEN_UNCHANGED"
        ),
    ),

    (
        "exactly_one_test_fd001_file",
        len(
            test_matches_628hc6a9a2
        )
        ==
        1,
    ),

    (
        "exactly_one_rul_fd001_file",
        len(
            rul_matches_628hc6a9a2
        )
        ==
        1,
    ),

    (
        "test_schema_matches_locked_fd001_raw_schema",
        schema_exact_628hc6a9a2,
    ),

    (
        "test_raw_finite",
        all_numeric_finite_628hc6a9a2,
    ),

    (
        "test_engine_count_matches_rul_count",
        engine_rul_count_match_628hc6a9a2,
    ),

    (
        "test_rul_finite",
        rul_finite_628hc6a9a2,
    ),

    (
        "test_rul_nonnegative",
        rul_nonnegative_628hc6a9a2,
    ),

    (
        "retained_sensors_present",
        retained_sensors_present_628hc6a9a2,
    ),

    (
        "test_cycles_positive",
        cycles_positive_628hc6a9a2,
    ),

    (
        "frozen_threshold_unchanged",
        threshold_unchanged_628hc6a9a2,
    ),

    (
        "no_normalization_in_this_step",
        True,
    ),

    (
        "no_window_construction_in_this_step",
        True,
    ),

    (
        "no_model_execution_in_this_step",
        True,
    ),

    (
        "no_mc_dropout_execution_in_this_step",
        True,
    ),

    (
        "no_test_scoring_in_this_step",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]


C6_A9_A2_AUDIT = pd.DataFrame(
    audit_rows_628hc6a9a2,
    columns=[
        "check",
        "passed",
    ],
)


all_passed_628hc6a9a2 = bool(
    C6_A9_A2_AUDIT[
        "passed"
    ].all()
)


# --------------------------------------------------------------------------------------------
# 7. Output
# --------------------------------------------------------------------------------------------

print(
    "\n--- FD001 TEST READ CONTRACT ---"
)


print(
    "TEST path                  :",
    FD001_TEST_PATH_628HC6A9A2,
)

print(
    "RUL path                   :",
    FD001_RUL_PATH_628HC6A9A2,
)

print(
    "TEST raw shape             :",
    test_raw_628hc6a9a2.shape,
)

print(
    "TEST engine count          :",
    test_engine_count_628hc6a9a2,
)

print(
    "Official RUL count         :",
    len(
        test_rul_628hc6a9a2
    ),
)

print(
    "Min rows per engine        :",
    minimum_engine_history_628hc6a9a2,
)

print(
    "Max rows per engine        :",
    int(
        rows_per_engine_628hc6a9a2.max()
    ),
)

print(
    "Min final cycle            :",
    int(
        last_cycle_by_engine_628hc6a9a2.min()
    ),
)

print(
    "Max final cycle            :",
    int(
        last_cycle_by_engine_628hc6a9a2.max()
    ),
)

print(
    "RUL min / median / max     :",
    float(
        np.min(
            test_rul_628hc6a9a2
        )
    ),
    float(
        np.median(
            test_rul_628hc6a9a2
        )
    ),
    float(
        np.max(
            test_rul_628hc6a9a2
        )
    ),
)

print(
    "Frozen threshold           :",
    frozen_threshold_after_628hc6a9a2,
)


print(
    "\n--- C6-A9-A2 AUDIT ---"
)

display(
    C6_A9_A2_AUDIT
)


print(
    "\n"
    +
    "=" * 116
)

print(
    "STEP 6.28H-C6-A9-A2 FINAL SUMMARY"
)

print(
    "=" * 116
)

print(
    "Official FD001 TEST features read : True"
)

print(
    "Official FD001 RUL labels read    : True"
)

print(
    "TEST normalized                   : False"
)

print(
    "TEST windows built                : False"
)

print(
    "Model executed                    : False"
)

print(
    "MC dropout executed               : False"
)

print(
    "TEST scored                       : False"
)

print(
    "Model retrained                   : False"
)

print(
    "Threshold tuned/modified          : False"
)

print(
    "Files written                     : False"
)

print(
    "All C6-A9-A2 checks passed        :",
    all_passed_628hc6a9a2,
)


if not all_passed_628hc6a9a2:

    failed_628hc6a9a2 = (
        C6_A9_A2_AUDIT
        .loc[
            ~C6_A9_A2_AUDIT[
                "passed"
            ],
            "check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "C6-A9-A2 failed: "
        +
        ", ".join(
            failed_628hc6a9a2
        )
    )


print(
    "\nPASS — official FD001 TEST sources were resolved "
    "by exact filename and read once."
)

print(
    "Next: build the frozen-normalized endpoint TEST windows, "
    "still without model execution or scoring."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A2 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A2-R1 — FD001 FILESYSTEM ROOT CORRECTION
# Path discovery only. No TEST/RUL file contents are opened.
# ============================================================================================

from pathlib import Path

print("=" * 110)
print("STEP 6.28H-C6-A9-A2-R1 — FD001 FILESYSTEM ROOT CORRECTION")
print("=" * 110)

notebook_cwd = Path.cwd().resolve()
repository_root = notebook_cwd.parent

print("\nNotebook working directory :", notebook_cwd)
print("Repository search root     :", repository_root)

test_matches = sorted(
    p.resolve()
    for p in repository_root.rglob("*")
    if p.is_file()
    and p.name.lower() == "test_fd001.txt"
)

rul_matches = sorted(
    p.resolve()
    for p in repository_root.rglob("*")
    if p.is_file()
    and p.name.lower() == "rul_fd001.txt"
)

print("\n--- EXACT FD001 FILE MATCHES ---")

print("TEST matches :", len(test_matches))
for p in test_matches:
    print("   ", p)

print("RUL matches  :", len(rul_matches))
for p in rul_matches:
    print("   ", p)

print("\n--- SAFETY ---")
print("TEST file contents read       : False")
print("Official RUL contents read    : False")
print("Model executed                : False")
print("MC dropout executed           : False")
print("TEST scored                   : False")
print("Threshold modified            : False")
print("Files written                 : False")

print("\n=== END STEP 6.28H-C6-A9-A2-R1 ===")

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A2-R2 — FD001 OFFICIAL TEST FEATURE ONE-TIME READ + INPUT CONTRACT LOCK
#
# Reads test_FD001.txt exactly once.
# DOES NOT open RUL_FD001.txt yet.
# DOES NOT execute model / MC dropout.
# DOES NOT score TEST.
# DOES NOT modify frozen policy.
# ============================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C6-A9-A2-R2 — FD001 OFFICIAL TEST FEATURE ONE-TIME READ + INPUT CONTRACT LOCK")
print("=" * 118)

# --------------------------------------------------------------------------------------------
# 0) Rerun guard
# --------------------------------------------------------------------------------------------
if globals().get("FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2", False):
    raise RuntimeError(
        "FD001 TEST features were already read by C6-A9-A2-R2. "
        "Refusing to reopen the file."
    )

# --------------------------------------------------------------------------------------------
# 1) Exact resolved paths from C6-A9-A2-R1
# --------------------------------------------------------------------------------------------
test_path = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\test_FD001.txt"
).resolve()

rul_path = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\RUL_FD001.txt"
).resolve()

if not test_path.is_file():
    raise FileNotFoundError(
        f"FD001 TEST feature file not found: {test_path}"
    )

if not rul_path.is_file():
    raise FileNotFoundError(
        f"FD001 official RUL file not found: {rul_path}"
    )

FD001_TEST_FEATURE_PATH_628HC6A9A2R2 = test_path
FD001_OFFICIAL_RUL_PATH_628HC6A9A2R2 = rul_path

# --------------------------------------------------------------------------------------------
# 2) Exact raw-column contract from already-locked FD001 DEV object
# --------------------------------------------------------------------------------------------
if "FD001_DEV_RAW_628HC6A3R1" not in globals():
    raise RuntimeError(
        "Required object FD001_DEV_RAW_628HC6A3R1 is missing."
    )

dev_raw_contract = FD001_DEV_RAW_628HC6A3R1

if not isinstance(dev_raw_contract, pd.DataFrame):
    raise TypeError(
        "FD001_DEV_RAW_628HC6A3R1 must be a pandas DataFrame."
    )

expected_columns = list(
    dev_raw_contract.columns
)

if len(expected_columns) != 26:
    raise RuntimeError(
        "Expected 26 raw FD001 columns from DEV contract; "
        f"found {len(expected_columns)}."
    )

unit_col = expected_columns[0]
cycle_col = expected_columns[1]

# --------------------------------------------------------------------------------------------
# 3) Snapshot frozen policy before TEST read
# --------------------------------------------------------------------------------------------
threshold_before = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

def sha256_array_628hc6a9a2r2(x):
    a = np.ascontiguousarray(
        np.asarray(x)
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()

phase_edges_sha_before = sha256_array_628hc6a9a2r2(
    FD001_PHASE_EDGES_628HC6A7A2A2
)

g1_sha_before = sha256_array_628hc6a9a2r2(
    FD001_G1_REFERENCE_628HC6A7A2A2
)

g2_values_sha_before = sha256_array_628hc6a9a2r2(
    FD001_G2_REFERENCE_VALUES_628HC6A7A2A2
)

g2_weights_sha_before = sha256_array_628hc6a9a2r2(
    FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2
)

# --------------------------------------------------------------------------------------------
# 4) ONE-TIME TEST FEATURE READ
#    RUL_FD001.txt remains unopened.
# --------------------------------------------------------------------------------------------
test_raw = pd.read_csv(
    test_path,
    sep=r"\s+",
    header=None,
    engine="python",
)

if test_raw.shape[1] != len(expected_columns):
    raise RuntimeError(
        "FD001 TEST raw column-count mismatch: "
        f"expected {len(expected_columns)}, "
        f"found {test_raw.shape[1]}."
    )

test_raw.columns = expected_columns

test_raw = test_raw.apply(
    pd.to_numeric,
    errors="raise",
)

if test_raw.empty:
    raise RuntimeError(
        "FD001 TEST feature table is empty."
    )

test_values = test_raw.to_numpy(
    dtype=np.float64
)

if not np.isfinite(
    test_values
).all():
    raise RuntimeError(
        "FD001 TEST features contain non-finite values."
    )

if test_raw[
    [unit_col, cycle_col]
].duplicated().any():
    raise RuntimeError(
        "Duplicate (unit, cycle) rows found in FD001 TEST."
    )

FD001_TEST_RAW_628HC6A9A2R2 = (
    test_raw.copy(
        deep=True
    )
)

FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2 = True

# --------------------------------------------------------------------------------------------
# 5) TEST-input-only metadata
# --------------------------------------------------------------------------------------------
test_unit_ids = np.sort(
    FD001_TEST_RAW_628HC6A9A2R2[
        unit_col
    ].unique()
)

test_engine_count = int(
    len(
        test_unit_ids
    )
)

test_row_count = int(
    len(
        FD001_TEST_RAW_628HC6A9A2R2
    )
)

terminal_cycles = (
    FD001_TEST_RAW_628HC6A9A2R2
    .groupby(
        unit_col,
        sort=True,
    )[cycle_col]
    .max()
    .to_numpy(
        dtype=np.int64
    )
)

# --------------------------------------------------------------------------------------------
# 6) Frozen-policy mutation audit
# --------------------------------------------------------------------------------------------
threshold_after = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

phase_edges_sha_after = sha256_array_628hc6a9a2r2(
    FD001_PHASE_EDGES_628HC6A7A2A2
)

g1_sha_after = sha256_array_628hc6a9a2r2(
    FD001_G1_REFERENCE_628HC6A7A2A2
)

g2_values_sha_after = sha256_array_628hc6a9a2r2(
    FD001_G2_REFERENCE_VALUES_628HC6A7A2A2
)

g2_weights_sha_after = sha256_array_628hc6a9a2r2(
    FD001_G2_REFERENCE_WEIGHTS_628HC6A7A2A2
)

threshold_unchanged = bool(
    threshold_after
    ==
    threshold_before
)

calibration_unchanged = bool(
    phase_edges_sha_after
    ==
    phase_edges_sha_before
    and
    g1_sha_after
    ==
    g1_sha_before
    and
    g2_values_sha_after
    ==
    g2_values_sha_before
    and
    g2_weights_sha_after
    ==
    g2_weights_sha_before
)

# --------------------------------------------------------------------------------------------
# 7) Audit
# --------------------------------------------------------------------------------------------
audit_rows = [
    (
        "test_feature_file_exists",
        test_path.is_file(),
    ),
    (
        "official_rul_file_exists_metadata_only",
        rul_path.is_file(),
    ),
    (
        "raw_column_contract_26",
        FD001_TEST_RAW_628HC6A9A2R2.shape[1] == 26,
    ),
    (
        "raw_columns_exactly_match_locked_dev",
        list(
            FD001_TEST_RAW_628HC6A9A2R2.columns
        )
        ==
        expected_columns,
    ),
    (
        "test_rows_positive",
        test_row_count > 0,
    ),
    (
        "test_engine_count_positive",
        test_engine_count > 0,
    ),
    (
        "test_features_finite",
        np.isfinite(
            FD001_TEST_RAW_628HC6A9A2R2.to_numpy(
                dtype=np.float64
            )
        ).all(),
    ),
    (
        "unit_cycle_pairs_unique",
        not FD001_TEST_RAW_628HC6A9A2R2[
            [unit_col, cycle_col]
        ].duplicated().any(),
    ),
    (
        "threshold_unchanged",
        threshold_unchanged,
    ),
    (
        "calibration_fingerprints_unchanged",
        calibration_unchanged,
    ),
    (
        "official_rul_contents_not_read",
        True,
    ),
    (
        "test_not_scored",
        True,
    ),
    (
        "no_model_execution",
        True,
    ),
    (
        "no_mc_dropout_execution",
        True,
    ),
    (
        "no_model_retraining",
        True,
    ),
    (
        "no_threshold_tuning_or_modification",
        True,
    ),
    (
        "no_files_written",
        True,
    ),
]

FD001_TEST_INPUT_AUDIT_628HC6A9A2R2 = (
    pd.DataFrame(
        audit_rows,
        columns=[
            "check",
            "passed",
        ],
    )
)

all_checks_passed = bool(
    FD001_TEST_INPUT_AUDIT_628HC6A9A2R2[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------
# 8) Reporting
# --------------------------------------------------------------------------------------------
print("\n--- RESOLVED SOURCES ---")
print(
    "TEST feature path :",
    FD001_TEST_FEATURE_PATH_628HC6A9A2R2,
)
print(
    "RUL path locked   :",
    FD001_OFFICIAL_RUL_PATH_628HC6A9A2R2,
)
print(
    "RUL contents read : False"
)

print("\n--- FD001 TEST FEATURE CONTRACT ---")
print(
    "Raw TEST shape                 :",
    FD001_TEST_RAW_628HC6A9A2R2.shape,
)
print(
    "Raw column count               :",
    FD001_TEST_RAW_628HC6A9A2R2.shape[1],
)
print(
    "Raw columns match DEV          :",
    list(
        FD001_TEST_RAW_628HC6A9A2R2.columns
    )
    ==
    expected_columns,
)
print(
    "TEST engines                   :",
    test_engine_count,
)
print(
    "Min terminal observed cycle    :",
    int(
        terminal_cycles.min()
    ),
)
print(
    "Median terminal observed cycle :",
    float(
        np.median(
            terminal_cycles
        )
    ),
)
print(
    "Max terminal observed cycle    :",
    int(
        terminal_cycles.max()
    ),
)
print(
    "Finite feature values          : True"
)
print(
    "Unique (unit, cycle) rows      : True"
)

print("\n--- C6-A9-A2-R2 AUDIT ---")
print(
    FD001_TEST_INPUT_AUDIT_628HC6A9A2R2.to_string(
        index=False
    )
)

print("\n" + "=" * 118)
print("STEP 6.28H-C6-A9-A2-R2 FINAL SUMMARY")
print("=" * 118)

print(
    "TEST feature values read once      :",
    True,
)
print(
    "Official FD001 RUL contents read   :",
    False,
)
print(
    "TEST scored                        :",
    False,
)
print(
    "Model executed                     :",
    False,
)
print(
    "MC dropout executed                :",
    False,
)
print(
    "Model retrained                    :",
    False,
)
print(
    "Frozen threshold unchanged         :",
    threshold_unchanged,
)
print(
    "Frozen calibration unchanged       :",
    calibration_unchanged,
)
print(
    "Files written                      :",
    False,
)
print(
    "All C6-A9-A2-R2 checks passed      :",
    all_checks_passed,
)

if not all_checks_passed:
    raise RuntimeError(
        "C6-A9-A2-R2 contract audit failed. "
        "Do not proceed to TEST normalization/inference."
    )

print(
    "\nPASS — official FD001 TEST FEATURES are now locked in memory."
)
print(
    "The official RUL file has NOT been opened."
)
print(
    "Next: normalize/build TEST windows, run deterministic + "
    "50-pass MC dropout, apply the already-frozen reliability policy, "
    "and freeze predictions/accept decisions BEFORE opening RUL_FD001.txt."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A2-R2 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A3 — FD001 TEST EXECUTION + PRE-LABEL POLICY-DECISION FREEZE
#
# Uses the already-read FD001 TEST features in memory.
# Builds all 30-cycle TEST windows, runs deterministic + 50-pass MC dropout,
# applies the already-frozen FD001 reliability calibration and threshold,
# then freezes the 100 engine-endpoint predictions / risks / accept-abstain decisions.
#
# IMPORTANT:
#   - DOES NOT open RUL_FD001.txt.
#   - DOES NOT score TEST.
#   - DOES NOT tune/modify the threshold or calibration.
#   - DOES NOT retrain the model.
#   - DOES NOT write files.
# ============================================================================================

import hashlib
import time
import numpy as np
import pandas as pd
import torch

print("=" * 120)
print("STEP 6.28H-C6-A9-A3 — FD001 TEST EXECUTION + PRE-LABEL POLICY-DECISION FREEZE")
print("=" * 120)

# --------------------------------------------------------------------------------------------
# 0) One-way execution guard
# --------------------------------------------------------------------------------------------
if globals().get("FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3", False):
    raise RuntimeError(
        "C6-A9-A3 has already frozen the FD001 TEST predictions/decisions. "
        "Refusing to rerun the TEST model/MC stage."
    )

required_names = [
    "FD001_TEST_RAW_628HC6A9A2R2",
    "FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2",
    "normalize_fd001_628hc6a8a5",
    "build_windows_fd001_628hc6a8a5",
    "MODEL_FD001_628HC6A6A1",
    "mc_dropout_predict_628hc0",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "fd001_phase_adjust_628hc6a7a2a2",
    "fd001_g1_risk_628hc6a7a2a2",
    "fd001_final_risk_628hc6a7a2a2",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Missing required frozen object(s): "
        + ", ".join(missing_names)
    )

if not bool(FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2):
    raise RuntimeError(
        "The one-time TEST-feature read contract has not passed."
    )

# --------------------------------------------------------------------------------------------
# 1) Lock source objects
# --------------------------------------------------------------------------------------------
test_raw = FD001_TEST_RAW_628HC6A9A2R2
model = MODEL_FD001_628HC6A6A1

threshold = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

if not isinstance(test_raw, pd.DataFrame):
    raise TypeError(
        "FD001_TEST_RAW_628HC6A9A2R2 must be a pandas DataFrame."
    )

if test_raw.shape != (13096, 26):
    raise RuntimeError(
        "Unexpected FD001 TEST raw shape: "
        f"{test_raw.shape}; expected (13096, 26)."
    )

unit_col = test_raw.columns[0]
cycle_col = test_raw.columns[1]

# --------------------------------------------------------------------------------------------
# 2) Model-state fingerprint helper
# --------------------------------------------------------------------------------------------
def model_sha256_628hc6a9a3(model_obj):
    h = hashlib.sha256()

    for key, value in model_obj.state_dict().items():
        h.update(str(key).encode("utf-8"))

        a = (
            value
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            np.ascontiguousarray(a).tobytes(order="C")
        )

    return h.hexdigest()

model_sha_before = model_sha256_628hc6a9a3(
    model
)

training_state_before = bool(
    model.training
)

threshold_before = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

# --------------------------------------------------------------------------------------------
# 3) Frozen normalization
# --------------------------------------------------------------------------------------------
t0 = time.time()

normalized_test = normalize_fd001_628hc6a8a5(
    test_raw.copy(deep=True)
)

if not isinstance(normalized_test, pd.DataFrame):
    raise TypeError(
        "normalize_fd001_628hc6a8a5 did not return a DataFrame."
    )

if len(normalized_test) != len(test_raw):
    raise RuntimeError(
        "Normalization changed TEST row count."
    )

# --------------------------------------------------------------------------------------------
# 4) Locked 30-cycle window bridge
# --------------------------------------------------------------------------------------------
window_out = build_windows_fd001_628hc6a8a5(
    normalized_test
)

if isinstance(window_out, np.ndarray):

    X_test_all = window_out

elif isinstance(window_out, (tuple, list)):

    window_candidates = [
        obj
        for obj in window_out
        if isinstance(obj, np.ndarray)
        and obj.ndim == 3
    ]

    if len(window_candidates) != 1:
        raise RuntimeError(
            "Could not uniquely resolve the TEST window tensor "
            "from build_windows_fd001_628hc6a8a5 output."
        )

    X_test_all = window_candidates[0]

else:

    raise TypeError(
        "Unexpected build_windows_fd001_628hc6a8a5 return type: "
        f"{type(window_out).__name__}"
    )

X_test_all = np.asarray(
    X_test_all,
    dtype=np.float32,
)

engine_row_counts = (
    test_raw
    .groupby(
        unit_col,
        sort=True,
    )
    .size()
)

if len(engine_row_counts) != 100:
    raise RuntimeError(
        "Expected 100 FD001 TEST engines."
    )

if bool((engine_row_counts < 30).any()):
    raise RuntimeError(
        "At least one TEST engine has fewer than 30 observed cycles."
    )

windows_per_engine = (
    engine_row_counts
    - 30
    + 1
).astype(np.int64)

expected_window_count = int(
    windows_per_engine.sum()
)

if X_test_all.shape != (
    expected_window_count,
    30,
    17,
):
    raise RuntimeError(
        "TEST window shape mismatch: "
        f"got {X_test_all.shape}; "
        f"expected ({expected_window_count}, 30, 17)."
    )

if not np.isfinite(X_test_all).all():
    raise RuntimeError(
        "TEST windows contain non-finite values."
    )

# Endpoint of each engine in grouped sliding-window order.
endpoint_indices = (
    np.cumsum(
        windows_per_engine.to_numpy(
            dtype=np.int64
        )
    )
    - 1
)

endpoint_unit_ids = (
    engine_row_counts.index
    .to_numpy()
)

endpoint_cycles = (
    test_raw
    .groupby(
        unit_col,
        sort=True,
    )[cycle_col]
    .max()
    .to_numpy(
        dtype=np.int64
    )
)

if endpoint_indices.shape != (100,):
    raise RuntimeError(
        "Expected exactly 100 endpoint window indices."
    )

# --------------------------------------------------------------------------------------------
# 5) Deterministic inference
# --------------------------------------------------------------------------------------------
device = next(
    model.parameters()
).device

batch_size = 512
n_test_windows = len(X_test_all)

deterministic_all = np.empty(
    n_test_windows,
    dtype=np.float64,
)

model.eval()

with torch.no_grad():

    for start in range(
        0,
        n_test_windows,
        batch_size,
    ):

        end = min(
            start + batch_size,
            n_test_windows,
        )

        xb = torch.as_tensor(
            X_test_all[start:end],
            dtype=torch.float32,
            device=device,
        )

        pred = (
            model(xb)
            .reshape(-1)
            .detach()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        deterministic_all[start:end] = pred

# --------------------------------------------------------------------------------------------
# 6) Frozen 50-pass MC dropout
#    Helper contract = (mean, std, runtime)
# --------------------------------------------------------------------------------------------
mc_out = mc_dropout_predict_628hc0(
    X_test_all,
    model,
    passes=50,
    batch_size=512,
    seed=42,
)

if (
    not isinstance(mc_out, (tuple, list))
    or len(mc_out) != 3
):
    raise RuntimeError(
        "mc_dropout_predict_628hc0 must return "
        "(mc_mean, mc_std, runtime)."
    )

mc_mean_all = np.asarray(
    mc_out[0],
    dtype=np.float64,
).reshape(-1)

mc_std_all = np.asarray(
    mc_out[1],
    dtype=np.float64,
).reshape(-1)

mc_runtime = float(
    mc_out[2]
)

if (
    deterministic_all.shape != (n_test_windows,)
    or mc_mean_all.shape != (n_test_windows,)
    or mc_std_all.shape != (n_test_windows,)
):
    raise RuntimeError(
        "TEST deterministic/MC output shape mismatch."
    )

if not (
    np.isfinite(deterministic_all).all()
    and np.isfinite(mc_mean_all).all()
    and np.isfinite(mc_std_all).all()
):
    raise RuntimeError(
        "TEST deterministic/MC outputs contain non-finite values."
    )

if bool((mc_std_all < 0.0).any()):
    raise RuntimeError(
        "TEST MC standard deviation contains negative values."
    )

# --------------------------------------------------------------------------------------------
# 7) Apply the already-frozen FD001 reliability construction
# --------------------------------------------------------------------------------------------
phase_all = np.asarray(
    fd001_phase_adjust_628hc6a7a2a2(
        deterministic_all,
        mc_std_all,
    ),
    dtype=np.float64,
).reshape(-1)

g1_all = np.asarray(
    fd001_g1_risk_628hc6a7a2a2(
        phase_all
    ),
    dtype=np.float64,
).reshape(-1)

risk_all = np.asarray(
    fd001_final_risk_628hc6a7a2a2(
        g1_all
    ),
    dtype=np.float64,
).reshape(-1)

for name, arr in [
    ("phase", phase_all),
    ("g1", g1_all),
    ("final risk", risk_all),
]:

    if arr.shape != (n_test_windows,):
        raise RuntimeError(
            f"{name} output shape mismatch: {arr.shape}"
        )

    if not np.isfinite(arr).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )

    if (
        float(arr.min()) < 0.0
        or float(arr.max()) > 1.0
    ):
        raise RuntimeError(
            f"{name} fell outside [0, 1]."
        )

accept_all = (
    risk_all
    <= threshold
)

abstain_all = ~accept_all

# --------------------------------------------------------------------------------------------
# 8) Official engine endpoints
# --------------------------------------------------------------------------------------------
deterministic_endpoint = deterministic_all[
    endpoint_indices
]

mc_mean_endpoint = mc_mean_all[
    endpoint_indices
]

mc_std_endpoint = mc_std_all[
    endpoint_indices
]

phase_endpoint = phase_all[
    endpoint_indices
]

g1_endpoint = g1_all[
    endpoint_indices
]

risk_endpoint = risk_all[
    endpoint_indices
]

accept_endpoint = accept_all[
    endpoint_indices
]

abstain_endpoint = abstain_all[
    endpoint_indices
]

FD001_TEST_ENDPOINT_POLICY_628HC6A9A3 = pd.DataFrame(
    {
        "unit_id":
            endpoint_unit_ids,

        "observed_terminal_cycle":
            endpoint_cycles,

        "deterministic_prediction":
            deterministic_endpoint,

        "mc_mean":
            mc_mean_endpoint,

        "mc_std":
            mc_std_endpoint,

        "phase_adjusted":
            phase_endpoint,

        "g1_risk":
            g1_endpoint,

        "reliability_risk":
            risk_endpoint,

        "accepted":
            accept_endpoint,

        "abstained":
            abstain_endpoint,
    }
)

if FD001_TEST_ENDPOINT_POLICY_628HC6A9A3.shape != (
    100,
    10,
):
    raise RuntimeError(
        "Endpoint policy table must contain exactly 100 rows."
    )

# --------------------------------------------------------------------------------------------
# 9) Freeze all PRE-LABEL TEST artifacts in memory
# --------------------------------------------------------------------------------------------
FD001_TEST_NORMALIZED_628HC6A9A3 = (
    normalized_test.copy(deep=True)
)

FD001_TEST_WINDOWS_628HC6A9A3 = (
    X_test_all.copy()
)

FD001_TEST_DETERMINISTIC_ALL_628HC6A9A3 = (
    deterministic_all.copy()
)

FD001_TEST_MC_MEAN_ALL_628HC6A9A3 = (
    mc_mean_all.copy()
)

FD001_TEST_MC_STD_ALL_628HC6A9A3 = (
    mc_std_all.copy()
)

FD001_TEST_PHASE_ADJUSTED_ALL_628HC6A9A3 = (
    phase_all.copy()
)

FD001_TEST_G1_RISK_ALL_628HC6A9A3 = (
    g1_all.copy()
)

FD001_TEST_RELIABILITY_RISK_ALL_628HC6A9A3 = (
    risk_all.copy()
)

FD001_TEST_ENDPOINT_INDICES_628HC6A9A3 = (
    endpoint_indices.copy()
)

FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3 = True

# --------------------------------------------------------------------------------------------
# 10) Model / policy mutation audit
# --------------------------------------------------------------------------------------------
if training_state_before:
    model.train()
else:
    model.eval()

model_sha_after = model_sha256_628hc6a9a3(
    model
)

training_state_after = bool(
    model.training
)

threshold_after = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

model_sha_unchanged = bool(
    model_sha_after
    ==
    model_sha_before
)

training_state_restored = bool(
    training_state_after
    ==
    training_state_before
)

threshold_unchanged = bool(
    threshold_after
    ==
    threshold_before
)

total_runtime = float(
    time.time() - t0
)

# --------------------------------------------------------------------------------------------
# 11) Immutable endpoint fingerprints
# --------------------------------------------------------------------------------------------
def sha256_array_628hc6a9a3(x):

    a = np.ascontiguousarray(
        np.asarray(x)
    )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


FD001_TEST_PRELABEL_FINGERPRINTS_628HC6A9A3 = {

    "deterministic_endpoint":
        sha256_array_628hc6a9a3(
            deterministic_endpoint
        ),

    "mc_mean_endpoint":
        sha256_array_628hc6a9a3(
            mc_mean_endpoint
        ),

    "mc_std_endpoint":
        sha256_array_628hc6a9a3(
            mc_std_endpoint
        ),

    "risk_endpoint":
        sha256_array_628hc6a9a3(
            risk_endpoint
        ),

    "accept_endpoint":
        sha256_array_628hc6a9a3(
            accept_endpoint.astype(
                np.uint8
            )
        ),
}

# --------------------------------------------------------------------------------------------
# 12) Audit
# --------------------------------------------------------------------------------------------
audit_rows = [

    (
        "test_features_read_once_contract",
        bool(
            FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2
        ),
    ),

    (
        "raw_test_shape_13096x26",
        test_raw.shape == (13096, 26),
    ),

    (
        "test_engine_count_100",
        len(engine_row_counts) == 100,
    ),

    (
        "all_engines_have_at_least_30_cycles",
        bool(
            (
                engine_row_counts >= 30
            ).all()
        ),
    ),

    (
        "window_tensor_shape_exact",
        X_test_all.shape
        ==
        (
            expected_window_count,
            30,
            17,
        ),
    ),

    (
        "window_tensor_finite",
        np.isfinite(
            X_test_all
        ).all(),
    ),

    (
        "endpoint_count_100",
        len(endpoint_indices) == 100,
    ),

    (
        "deterministic_finite",
        np.isfinite(
            deterministic_all
        ).all(),
    ),

    (
        "mc_mean_finite",
        np.isfinite(
            mc_mean_all
        ).all(),
    ),

    (
        "mc_std_finite_nonnegative",
        bool(
            np.isfinite(
                mc_std_all
            ).all()
            and
            (
                mc_std_all >= 0.0
            ).all()
        ),
    ),

    (
        "phase_in_0_1",
        bool(
            np.isfinite(
                phase_all
            ).all()
            and
            (
                phase_all >= 0.0
            ).all()
            and
            (
                phase_all <= 1.0
            ).all()
        ),
    ),

    (
        "g1_in_0_1",
        bool(
            np.isfinite(
                g1_all
            ).all()
            and
            (
                g1_all >= 0.0
            ).all()
            and
            (
                g1_all <= 1.0
            ).all()
        ),
    ),

    (
        "risk_in_0_1",
        bool(
            np.isfinite(
                risk_all
            ).all()
            and
            (
                risk_all >= 0.0
            ).all()
            and
            (
                risk_all <= 1.0
            ).all()
        ),
    ),

    (
        "endpoint_policy_rows_100",
        len(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3
        )
        == 100,
    ),

    (
        "endpoint_accept_abstain_complement",
        bool(
            np.array_equal(
                abstain_endpoint,
                ~accept_endpoint,
            )
        ),
    ),

    (
        "model_state_sha256_unchanged",
        model_sha_unchanged,
    ),

    (
        "model_training_state_restored",
        training_state_restored,
    ),

    (
        "threshold_unchanged",
        threshold_unchanged,
    ),

    (
        "official_rul_contents_not_read",
        True,
    ),

    (
        "test_not_scored",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

FD001_TEST_EXECUTION_AUDIT_628HC6A9A3 = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)

all_checks_passed = bool(
    FD001_TEST_EXECUTION_AUDIT_628HC6A9A3[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------------
# 13) Reporting
# --------------------------------------------------------------------------------------------
print("\n--- TEST WINDOW / EXECUTION CONTRACT ---")

print(
    "Raw TEST rows              :",
    len(test_raw),
)

print(
    "TEST engines               :",
    len(engine_row_counts),
)

print(
    "All 30-cycle windows       :",
    X_test_all.shape,
)

print(
    "Endpoint windows           :",
    len(endpoint_indices),
)

print(
    "Model device               :",
    device,
)

print(
    "MC passes                  : 50"
)

print(
    "MC batch size              : 512"
)

print(
    "MC seed                    : 42"
)

print(
    "MC runtime (sec)           :",
    mc_runtime,
)

print(
    "Total execution time (sec) :",
    total_runtime,
)

print("\n--- PRE-LABEL TEST ENDPOINT POLICY ---")

endpoint_summary = pd.DataFrame(
    {
        "metric": [
            "engines",
            "accepted",
            "abstained",
            "realized_coverage",
            "prediction_median",
            "mc_std_median",
            "risk_median",
            "risk_mean",
            "risk_max",
        ],

        "value": [
            100,

            int(
                accept_endpoint.sum()
            ),

            int(
                abstain_endpoint.sum()
            ),

            float(
                accept_endpoint.mean()
            ),

            float(
                np.median(
                    deterministic_endpoint
                )
            ),

            float(
                np.median(
                    mc_std_endpoint
                )
            ),

            float(
                np.median(
                    risk_endpoint
                )
            ),

            float(
                np.mean(
                    risk_endpoint
                )
            ),

            float(
                np.max(
                    risk_endpoint
                )
            ),
        ],
    }
)

print(
    endpoint_summary.to_string(
        index=False
    )
)

print("\n--- ENDPOINT DECISION PREVIEW ---")

print(
    FD001_TEST_ENDPOINT_POLICY_628HC6A9A3[
        [
            "unit_id",
            "observed_terminal_cycle",
            "deterministic_prediction",
            "mc_std",
            "reliability_risk",
            "accepted",
            "abstained",
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)

print("\n--- C6-A9-A3 AUDIT ---")

print(
    FD001_TEST_EXECUTION_AUDIT_628HC6A9A3.to_string(
        index=False
    )
)

print("\n" + "=" * 120)
print("STEP 6.28H-C6-A9-A3 FINAL SUMMARY")
print("=" * 120)

print(
    "All TEST windows processed          :",
    n_test_windows,
)

print(
    "Official endpoint predictions       :",
    len(
        deterministic_endpoint
    ),
)

print(
    "Endpoint accepted / abstained       :",
    int(
        accept_endpoint.sum()
    ),
    "/",
    int(
        abstain_endpoint.sum()
    ),
)

print(
    "Endpoint realized coverage          :",
    float(
        accept_endpoint.mean()
    ),
)

print(
    "Frozen threshold                    :",
    threshold,
)

print(
    "Model-state SHA-256 unchanged       :",
    model_sha_unchanged,
)

print(
    "Model training mode restored        :",
    training_state_restored,
)

print(
    "Threshold unchanged                 :",
    threshold_unchanged,
)

print(
    "Official FD001 RUL contents read    :",
    False,
)

print(
    "TEST scored                         :",
    False,
)

print(
    "Model retrained                     :",
    False,
)

print(
    "Threshold/calibration tuned         :",
    False,
)

print(
    "Files written                       :",
    False,
)

print(
    "All C6-A9-A3 checks passed          :",
    all_checks_passed,
)

if not all_checks_passed:
    raise RuntimeError(
        "C6-A9-A3 audit failed. "
        "Do NOT open RUL_FD001.txt."
    )

print(
    "\nPASS — FD001 TEST predictions, uncertainties, risks, "
    "and accept/abstain decisions are now frozen in memory."
)

print(
    "RUL_FD001.txt has still NOT been opened."
)

print(
    "Next: verify the pre-label fingerprints once, then perform "
    "the single official RUL read and final TEST scoring with no policy changes."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A3 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A3-R1 — FD001 TEST EXECUTION + PRE-LABEL POLICY-DECISION FREEZE
# Corrects the raw-TEST -> canonical A5 bridge before window construction.
# Does NOT open RUL_FD001.txt and does NOT score TEST.
# ============================================================================================

import hashlib
import time
import numpy as np
import pandas as pd
import torch

print("=" * 120)
print("STEP 6.28H-C6-A9-A3-R1 — FD001 TEST EXECUTION + PRE-LABEL POLICY-DECISION FREEZE")
print("=" * 120)

if globals().get("FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3R1", False):
    raise RuntimeError(
        "C6-A9-A3-R1 has already frozen the FD001 TEST decisions. Refusing to rerun."
    )

required = [
    "FD001_TEST_RAW_628HC6A9A2R2",
    "FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2",
    "FD001_DEV_RAW_628HC6A3R1",
    "FD001_DEV_CORRUPTION_BASE_628HC6A8A3",
    "normalize_fd001_628hc6a8a5",
    "build_windows_fd001_628hc6a8a5",
    "MODEL_FD001_628HC6A6A1",
    "mc_dropout_predict_628hc0",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "fd001_phase_adjust_628hc6a7a2a2",
    "fd001_g1_risk_628hc6a7a2a2",
    "fd001_final_risk_628hc6a7a2a2",
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing)
    )

if not bool(FD001_TEST_FEATURES_READ_ONCE_628HC6A9A2R2):
    raise RuntimeError(
        "The one-time FD001 TEST feature-read contract is not locked."
    )

test_raw = FD001_TEST_RAW_628HC6A9A2R2
dev_raw = FD001_DEV_RAW_628HC6A3R1
dev_base = FD001_DEV_CORRUPTION_BASE_628HC6A8A3
model = MODEL_FD001_628HC6A6A1

threshold = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

if (
    not isinstance(test_raw, pd.DataFrame)
    or test_raw.shape != (13096, 26)
):
    raise RuntimeError(
        f"Unexpected TEST raw contract: "
        f"{type(test_raw).__name__}, "
        f"{getattr(test_raw, 'shape', None)}"
    )

if (
    not isinstance(dev_raw, pd.DataFrame)
    or not isinstance(dev_base, pd.DataFrame)
):
    raise RuntimeError(
        "Locked DEV raw/base objects are not DataFrames."
    )

# --------------------------------------------------------------------------------------------
# 1) Recover exact raw-DEV -> canonical-DEV column bridge.
# --------------------------------------------------------------------------------------------
if "regime_id" not in dev_base.columns:
    raise RuntimeError(
        "Canonical FD001 DEV base does not contain regime_id."
    )

canonical_nonregime_cols = [
    c for c in dev_base.columns
    if c != "regime_id"
]

if len(canonical_nonregime_cols) != 26:
    raise RuntimeError(
        f"Expected 26 canonical non-regime columns, "
        f"found {len(canonical_nonregime_cols)}."
    )

if len(dev_raw) != len(dev_base):
    raise RuntimeError(
        "Locked DEV raw/base row counts differ; "
        "cannot recover bridge safely."
    )

raw_to_canonical = {}
used_raw_cols = set()

for canonical_col in canonical_nonregime_cols:

    if canonical_col in dev_raw.columns:

        lhs = dev_raw[
            canonical_col
        ].to_numpy()

        rhs = dev_base[
            canonical_col
        ].to_numpy()

        if np.array_equal(
            lhs,
            rhs,
            equal_nan=True,
        ):
            raw_to_canonical[
                canonical_col
            ] = canonical_col

            used_raw_cols.add(
                canonical_col
            )

            continue

    candidates = []

    rhs = dev_base[
        canonical_col
    ].to_numpy()

    for raw_col in dev_raw.columns:

        if raw_col in used_raw_cols:
            continue

        lhs = dev_raw[
            raw_col
        ].to_numpy()

        if np.array_equal(
            lhs,
            rhs,
            equal_nan=True,
        ):
            candidates.append(
                raw_col
            )

    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely recover raw->canonical "
            f"mapping for {canonical_col!r}; "
            f"candidates={candidates}."
        )

    raw_col = candidates[0]

    raw_to_canonical[
        raw_col
    ] = canonical_col

    used_raw_cols.add(
        raw_col
    )

if set(raw_to_canonical.keys()) != set(dev_raw.columns):
    raise RuntimeError(
        "Recovered raw->canonical bridge does not cover "
        "all 26 DEV raw columns."
    )

if set(raw_to_canonical.keys()) != set(test_raw.columns):
    raise RuntimeError(
        "TEST raw columns do not match the exact locked "
        "DEV raw columns."
    )

regime_values = pd.unique(
    dev_base[
        "regime_id"
    ]
)

if len(regime_values) != 1:
    raise RuntimeError(
        f"Expected single-regime FD001 canonical base; "
        f"found {regime_values.tolist()}."
    )

regime_value = regime_values[0]

test_base = (
    test_raw
    .rename(
        columns=raw_to_canonical
    )
    .copy(
        deep=True
    )
)

test_base[
    "regime_id"
] = regime_value

test_base = (
    test_base
    .loc[
        :,
        dev_base.columns
    ]
    .copy()
)

if test_base.shape != (13096, 27):
    raise RuntimeError(
        f"Canonical TEST base shape mismatch: "
        f"{test_base.shape}"
    )

if "unit_id" not in test_base.columns:
    raise RuntimeError(
        "Canonical TEST bridge still lacks unit_id."
    )

if "cycle" not in test_base.columns:
    raise RuntimeError(
        "Canonical TEST bridge still lacks cycle."
    )

print("\n--- RAW -> CANONICAL TEST BRIDGE ---")
print("Raw TEST shape       :", test_raw.shape)
print("Canonical TEST shape :", test_base.shape)
print("Identifier column    : unit_id")
print("Regime value         :", regime_value)
print("Bridge exact from DEV:", True)

# --------------------------------------------------------------------------------------------
# 2) Locked normalization.
# --------------------------------------------------------------------------------------------
t0 = time.time()

normalized_test = normalize_fd001_628hc6a8a5(
    test_base.copy(
        deep=True
    )
)

if not isinstance(
    normalized_test,
    pd.DataFrame,
):
    raise RuntimeError(
        "normalize_fd001_628hc6a8a5 did not "
        "return a DataFrame."
    )

if len(normalized_test) != 13096:
    raise RuntimeError(
        "Normalization changed TEST row count."
    )

if "unit_id" not in normalized_test.columns:
    raise RuntimeError(
        "Normalized TEST frame lacks unit_id."
    )

# --------------------------------------------------------------------------------------------
# 3) Build all 30-cycle TEST windows.
# --------------------------------------------------------------------------------------------
window_out = build_windows_fd001_628hc6a8a5(
    normalized_test
)

if isinstance(
    window_out,
    np.ndarray,
):
    X_test_all = window_out

elif isinstance(
    window_out,
    (tuple, list),
):

    window_candidates = [
        obj
        for obj in window_out
        if (
            isinstance(
                obj,
                np.ndarray,
            )
            and obj.ndim == 3
        )
    ]

    if len(window_candidates) != 1:
        raise RuntimeError(
            "Could not uniquely resolve the 3-D TEST "
            "window tensor from the A5 helper."
        )

    X_test_all = window_candidates[0]

else:
    raise RuntimeError(
        f"Unexpected A5 window-helper return type: "
        f"{type(window_out).__name__}"
    )

X_test_all = np.asarray(
    X_test_all,
    dtype=np.float32,
)

engine_counts = (
    test_base
    .groupby(
        "unit_id",
        sort=False,
    )
    .size()
    .astype(
        np.int64
    )
)

if len(engine_counts) != 100:
    raise RuntimeError(
        f"Expected 100 TEST engines; "
        f"found {len(engine_counts)}."
    )

if bool(
    (
        engine_counts < 30
    ).any()
):
    raise RuntimeError(
        "At least one TEST engine has fewer than 30 cycles."
    )

windows_per_engine = (
    engine_counts
    - 29
)

expected_windows = int(
    windows_per_engine.sum()
)

if expected_windows != 10196:
    raise RuntimeError(
        f"Expected 10,196 FD001 TEST windows "
        f"from the locked contract; got {expected_windows}."
    )

if X_test_all.shape != (
    10196,
    30,
    17,
):
    raise RuntimeError(
        f"TEST window tensor mismatch: "
        f"{X_test_all.shape}; "
        f"expected (10196, 30, 17)."
    )

if not np.isfinite(
    X_test_all
).all():
    raise RuntimeError(
        "TEST window tensor contains non-finite values."
    )

endpoint_indices = (
    np.cumsum(
        windows_per_engine.to_numpy()
    )
    - 1
)

endpoint_unit_ids = (
    engine_counts.index
    .to_numpy()
)

endpoint_cycles = (
    test_base
    .groupby(
        "unit_id",
        sort=False,
    )[
        "cycle"
    ]
    .max()
    .to_numpy(
        dtype=np.int64
    )
)

if endpoint_indices.shape != (100,):
    raise RuntimeError(
        "Endpoint-index contract is not exactly 100 engines."
    )

print("\n--- TEST WINDOW CONTRACT ---")
print(
    "All TEST windows :",
    X_test_all.shape,
)
print(
    "Endpoint windows :",
    len(endpoint_indices),
)

# --------------------------------------------------------------------------------------------
# 4) Model fingerprint.
# --------------------------------------------------------------------------------------------
def model_sha256_628hc6a9a3r1(
    model_obj
):

    h = hashlib.sha256()

    for key, value in (
        model_obj
        .state_dict()
        .items()
    ):

        h.update(
            str(key).encode(
                "utf-8"
            )
        )

        arr = (
            value
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            np.ascontiguousarray(
                arr
            ).tobytes(
                order="C"
            )
        )

    return h.hexdigest()


model_sha_before = (
    model_sha256_628hc6a9a3r1(
        model
    )
)

training_state_before = bool(
    model.training
)

threshold_before = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

device = next(
    model.parameters()
).device

# --------------------------------------------------------------------------------------------
# 5) Deterministic inference.
# --------------------------------------------------------------------------------------------
deterministic_all = np.empty(
    len(X_test_all),
    dtype=np.float64,
)

model.eval()

with torch.no_grad():

    for start in range(
        0,
        len(X_test_all),
        512,
    ):

        end = min(
            start + 512,
            len(X_test_all),
        )

        xb = torch.as_tensor(
            X_test_all[
                start:end
            ],
            dtype=torch.float32,
            device=device,
        )

        deterministic_all[
            start:end
        ] = (
            model(
                xb
            )
            .reshape(-1)
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float64
            )
        )

# --------------------------------------------------------------------------------------------
# 6) Frozen 50-pass MC dropout.
#    Exact historical helper contract = (mean, std, runtime).
# --------------------------------------------------------------------------------------------
mc_out = mc_dropout_predict_628hc0(
    X_test_all,
    model,
    passes=50,
    batch_size=512,
    seed=42,
)

if (
    not isinstance(
        mc_out,
        (tuple, list),
    )
    or len(mc_out) != 3
):
    raise RuntimeError(
        "mc_dropout_predict_628hc0 must return "
        "exactly (mean, std, runtime)."
    )

mc_mean_all = np.asarray(
    mc_out[0],
    dtype=np.float64,
).reshape(-1)

mc_std_all = np.asarray(
    mc_out[1],
    dtype=np.float64,
).reshape(-1)

mc_runtime = float(
    mc_out[2]
)

for name, arr in [
    (
        "deterministic",
        deterministic_all,
    ),
    (
        "mc_mean",
        mc_mean_all,
    ),
    (
        "mc_std",
        mc_std_all,
    ),
]:

    if arr.shape != (10196,):
        raise RuntimeError(
            f"{name} shape mismatch: "
            f"{arr.shape}"
        )

    if not np.isfinite(
        arr
    ).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )

if bool(
    (
        mc_std_all < 0.0
    ).any()
):
    raise RuntimeError(
        "MC standard deviation contains negative values."
    )

# --------------------------------------------------------------------------------------------
# 7) Already-frozen FD001 reliability construction.
# --------------------------------------------------------------------------------------------
phase_all = np.asarray(
    fd001_phase_adjust_628hc6a7a2a2(
        deterministic_all,
        mc_std_all,
    ),
    dtype=np.float64,
).reshape(-1)

g1_all = np.asarray(
    fd001_g1_risk_628hc6a7a2a2(
        phase_all
    ),
    dtype=np.float64,
).reshape(-1)

risk_all = np.asarray(
    fd001_final_risk_628hc6a7a2a2(
        g1_all
    ),
    dtype=np.float64,
).reshape(-1)

for name, arr in [
    (
        "phase_adjusted",
        phase_all,
    ),
    (
        "g1_risk",
        g1_all,
    ),
    (
        "final_risk",
        risk_all,
    ),
]:

    if arr.shape != (10196,):
        raise RuntimeError(
            f"{name} shape mismatch: "
            f"{arr.shape}"
        )

    if not np.isfinite(
        arr
    ).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )

    if (
        float(
            arr.min()
        ) < 0.0
        or float(
            arr.max()
        ) > 1.0
    ):
        raise RuntimeError(
            f"{name} is outside [0,1]."
        )

accept_all = (
    risk_all
    <= threshold
)

abstain_all = (
    ~accept_all
)

# --------------------------------------------------------------------------------------------
# 8) Freeze official 100 engine endpoints BEFORE any RUL read.
# --------------------------------------------------------------------------------------------
det_endpoint = deterministic_all[
    endpoint_indices
]

mc_mean_endpoint = mc_mean_all[
    endpoint_indices
]

mc_std_endpoint = mc_std_all[
    endpoint_indices
]

phase_endpoint = phase_all[
    endpoint_indices
]

g1_endpoint = g1_all[
    endpoint_indices
]

risk_endpoint = risk_all[
    endpoint_indices
]

accept_endpoint = accept_all[
    endpoint_indices
]

abstain_endpoint = abstain_all[
    endpoint_indices
]

FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1 = pd.DataFrame(
    {
        "unit_id":
            endpoint_unit_ids,

        "observed_terminal_cycle":
            endpoint_cycles,

        "deterministic_prediction":
            det_endpoint,

        "mc_mean":
            mc_mean_endpoint,

        "mc_std":
            mc_std_endpoint,

        "phase_adjusted":
            phase_endpoint,

        "g1_risk":
            g1_endpoint,

        "reliability_risk":
            risk_endpoint,

        "accepted":
            accept_endpoint,

        "abstained":
            abstain_endpoint,
    }
)

if (
    FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1.shape
    !=
    (
        100,
        10,
    )
):
    raise RuntimeError(
        f"Endpoint policy-table shape mismatch: "
        f"{FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1.shape}"
    )

# --------------------------------------------------------------------------------------------
# 9) In-memory PRE-LABEL freeze.
# --------------------------------------------------------------------------------------------
FD001_TEST_CANONICAL_BASE_628HC6A9A3R1 = (
    test_base.copy(
        deep=True
    )
)

FD001_TEST_NORMALIZED_628HC6A9A3R1 = (
    normalized_test.copy(
        deep=True
    )
)

FD001_TEST_WINDOWS_628HC6A9A3R1 = (
    X_test_all.copy()
)

FD001_TEST_DETERMINISTIC_ALL_628HC6A9A3R1 = (
    deterministic_all.copy()
)

FD001_TEST_MC_MEAN_ALL_628HC6A9A3R1 = (
    mc_mean_all.copy()
)

FD001_TEST_MC_STD_ALL_628HC6A9A3R1 = (
    mc_std_all.copy()
)

FD001_TEST_PHASE_ADJUSTED_ALL_628HC6A9A3R1 = (
    phase_all.copy()
)

FD001_TEST_G1_RISK_ALL_628HC6A9A3R1 = (
    g1_all.copy()
)

FD001_TEST_RELIABILITY_RISK_ALL_628HC6A9A3R1 = (
    risk_all.copy()
)

FD001_TEST_ENDPOINT_INDICES_628HC6A9A3R1 = (
    endpoint_indices.copy()
)

# --------------------------------------------------------------------------------------------
# 10) Pre-label fingerprints.
# --------------------------------------------------------------------------------------------
def sha256_array_628hc6a9a3r1(
    x
):

    arr = np.ascontiguousarray(
        np.asarray(
            x
        )
    )

    return hashlib.sha256(
        arr.tobytes(
            order="C"
        )
    ).hexdigest()


FD001_TEST_PRELABEL_FINGERPRINTS_628HC6A9A3R1 = {

    "endpoint_indices":
        sha256_array_628hc6a9a3r1(
            endpoint_indices.astype(
                np.int64
            )
        ),

    "deterministic_endpoint":
        sha256_array_628hc6a9a3r1(
            det_endpoint
        ),

    "mc_mean_endpoint":
        sha256_array_628hc6a9a3r1(
            mc_mean_endpoint
        ),

    "mc_std_endpoint":
        sha256_array_628hc6a9a3r1(
            mc_std_endpoint
        ),

    "risk_endpoint":
        sha256_array_628hc6a9a3r1(
            risk_endpoint
        ),

    "accept_endpoint":
        sha256_array_628hc6a9a3r1(
            accept_endpoint.astype(
                np.uint8
            )
        ),
}

# --------------------------------------------------------------------------------------------
# 11) Restore model state and verify frozen policy unchanged.
# --------------------------------------------------------------------------------------------
if training_state_before:
    model.train()
else:
    model.eval()

model_sha_after = (
    model_sha256_628hc6a9a3r1(
        model
    )
)

training_state_after = bool(
    model.training
)

threshold_after = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)

model_sha_unchanged = bool(
    model_sha_after
    ==
    model_sha_before
)

training_state_restored = bool(
    training_state_after
    ==
    training_state_before
)

threshold_unchanged = bool(
    threshold_after
    ==
    threshold_before
)

total_runtime = float(
    time.time()
    -
    t0
)

# --------------------------------------------------------------------------------------------
# 12) Audit.
# --------------------------------------------------------------------------------------------
audit_rows = [

    (
        "test_feature_read_contract_locked",
        True,
    ),

    (
        "raw_test_shape_13096x26",
        test_raw.shape
        ==
        (
            13096,
            26,
        ),
    ),

    (
        "raw_to_canonical_bridge_recovered_from_dev",
        True,
    ),

    (
        "canonical_test_shape_13096x27",
        test_base.shape
        ==
        (
            13096,
            27,
        ),
    ),

    (
        "canonical_test_has_unit_id",
        "unit_id"
        in test_base.columns,
    ),

    (
        "single_regime_preserved",
        test_base[
            "regime_id"
        ].nunique()
        ==
        1,
    ),

    (
        "normalized_test_rows_13096",
        len(
            normalized_test
        )
        ==
        13096,
    ),

    (
        "window_tensor_shape_10196x30x17",
        X_test_all.shape
        ==
        (
            10196,
            30,
            17,
        ),
    ),

    (
        "window_tensor_finite",
        np.isfinite(
            X_test_all
        ).all(),
    ),

    (
        "test_engine_count_100",
        len(
            engine_counts
        )
        ==
        100,
    ),

    (
        "endpoint_count_100",
        len(
            endpoint_indices
        )
        ==
        100,
    ),

    (
        "deterministic_shape_10196",
        deterministic_all.shape
        ==
        (
            10196,
        ),
    ),

    (
        "mc_mean_shape_10196",
        mc_mean_all.shape
        ==
        (
            10196,
        ),
    ),

    (
        "mc_std_shape_10196",
        mc_std_all.shape
        ==
        (
            10196,
        ),
    ),

    (
        "mc_std_nonnegative",
        bool(
            (
                mc_std_all
                >= 0.0
            ).all()
        ),
    ),

    (
        "risk_shape_10196",
        risk_all.shape
        ==
        (
            10196,
        ),
    ),

    (
        "risk_finite_in_0_1",
        bool(
            np.isfinite(
                risk_all
            ).all()
            and
            (
                risk_all
                >= 0.0
            ).all()
            and
            (
                risk_all
                <= 1.0
            ).all()
        ),
    ),

    (
        "endpoint_policy_rows_100",
        len(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1
        )
        ==
        100,
    ),

    (
        "endpoint_accept_abstain_complement",
        np.array_equal(
            abstain_endpoint,
            ~accept_endpoint,
        ),
    ),

    (
        "model_state_sha256_unchanged",
        model_sha_unchanged,
    ),

    (
        "model_training_mode_restored",
        training_state_restored,
    ),

    (
        "threshold_unchanged",
        threshold_unchanged,
    ),

    (
        "official_rul_contents_not_read",
        True,
    ),

    (
        "test_not_scored",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_tuning_or_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]

FD001_TEST_EXECUTION_AUDIT_628HC6A9A3R1 = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)

all_checks_passed = bool(
    FD001_TEST_EXECUTION_AUDIT_628HC6A9A3R1[
        "passed"
    ].all()
)

if not all_checks_passed:
    raise RuntimeError(
        "C6-A9-A3-R1 audit failed. "
        "Do NOT open RUL_FD001.txt."
    )

FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3R1 = True

# --------------------------------------------------------------------------------------------
# 13) Reporting.
# --------------------------------------------------------------------------------------------
print(
    "\n--- PRE-LABEL TEST ENDPOINT POLICY SUMMARY ---"
)

print(
    "TEST windows processed         :",
    len(
        X_test_all
    ),
)

print(
    "Official engine endpoints      :",
    len(
        endpoint_indices
    ),
)

print(
    "Accepted endpoints             :",
    int(
        accept_endpoint.sum()
    ),
)

print(
    "Abstained endpoints            :",
    int(
        abstain_endpoint.sum()
    ),
)

print(
    "Realized endpoint coverage     :",
    float(
        accept_endpoint.mean()
    ),
)

print(
    "Endpoint prediction median     :",
    float(
        np.median(
            det_endpoint
        )
    ),
)

print(
    "Endpoint MC-STD median         :",
    float(
        np.median(
            mc_std_endpoint
        )
    ),
)

print(
    "Endpoint risk median           :",
    float(
        np.median(
            risk_endpoint
        )
    ),
)

print(
    "Endpoint risk mean             :",
    float(
        np.mean(
            risk_endpoint
        )
    ),
)

print(
    "Endpoint risk max              :",
    float(
        np.max(
            risk_endpoint
        )
    ),
)

print(
    "MC runtime (sec)               :",
    mc_runtime,
)

print(
    "Total cell runtime (sec)       :",
    total_runtime,
)

print(
    "\n--- FIRST 10 FROZEN ENDPOINT DECISIONS ---"
)

print(
    FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
        [
            "unit_id",
            "observed_terminal_cycle",
            "deterministic_prediction",
            "mc_std",
            "reliability_risk",
            "accepted",
            "abstained",
        ]
    ]
    .head(
        10
    )
    .to_string(
        index=False
    )
)

print(
    "\n--- C6-A9-A3-R1 AUDIT ---"
)

print(
    FD001_TEST_EXECUTION_AUDIT_628HC6A9A3R1.to_string(
        index=False
    )
)

print(
    "\n"
    +
    "=" * 120
)

print(
    "STEP 6.28H-C6-A9-A3-R1 FINAL SUMMARY"
)

print(
    "=" * 120
)

print(
    "Raw->canonical TEST bridge exact       :",
    True,
)

print(
    "TEST window tensor                     :",
    X_test_all.shape,
)

print(
    "Endpoint decisions frozen              :",
    True,
)

print(
    "Frozen threshold                       :",
    threshold,
)

print(
    "Model-state SHA-256 unchanged          :",
    model_sha_unchanged,
)

print(
    "Model training mode restored           :",
    training_state_restored,
)

print(
    "Official FD001 RUL contents read       :",
    False,
)

print(
    "TEST scored                            :",
    False,
)

print(
    "Model retrained                        :",
    False,
)

print(
    "Threshold/calibration tuned            :",
    False,
)

print(
    "Files written                          :",
    False,
)

print(
    "All C6-A9-A3-R1 checks passed          :",
    all_checks_passed,
)

print(
    "\nPASS — FD001 TEST predictions, MC uncertainties, "
    "reliability risks, and endpoint accept/abstain "
    "decisions are frozen in memory BEFORE labels."
)

print(
    "RUL_FD001.txt has still NOT been opened."
)

print(
    "Next: verify these pre-label fingerprints, then "
    "perform the one-time official RUL read and final scoring."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A3-R1 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A4 — FD001 ONE-TIME OFFICIAL RUL READ + FINAL FROZEN-POLICY SCORING
#
# Contract:
#   1) verify PRE-LABEL fingerprints first
#   2) open official RUL_FD001.txt once
#   3) score ONLY already-frozen endpoint predictions/decisions
#   4) verify frozen policy objects remain unchanged
#
# No model execution.
# No MC dropout.
# No retraining.
# No threshold/calibration tuning.
# No file writing.
# ============================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 6.28H-C6-A9-A4 — FD001 ONE-TIME OFFICIAL RUL READ + FINAL FROZEN-POLICY SCORING")
print("=" * 120)


# --------------------------------------------------------------------------------------------
# 0) Required frozen objects
# --------------------------------------------------------------------------------------------

required = [
    "FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1",
    "FD001_TEST_ENDPOINT_INDICES_628HC6A9A3R1",
    "FD001_TEST_PRELABEL_FINGERPRINTS_628HC6A9A3R1",
    "FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3R1",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
    "FD001_NOMINAL_COVERAGE_628HC6A7A2A2",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required frozen object(s): "
        + ", ".join(missing)
    )


if not bool(
    FD001_TEST_POLICY_DECISIONS_FROZEN_628HC6A9A3R1
):
    raise RuntimeError(
        "Pre-label TEST policy decisions are not frozen. "
        "Do not read official RUL labels."
    )


if globals().get(
    "FD001_TEST_FINAL_SCORING_FROZEN_628HC6A9A4",
    False,
):
    raise RuntimeError(
        "C6-A9-A4 final scoring is already frozen. "
        "Refusing to score again."
    )


policy_pre = (
    FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1
    .copy(
        deep=True
    )
)


required_cols = [
    "unit_id",
    "observed_terminal_cycle",
    "deterministic_prediction",
    "mc_mean",
    "mc_std",
    "phase_adjusted",
    "g1_risk",
    "reliability_risk",
    "accepted",
    "abstained",
]


if list(
    policy_pre.columns
) != required_cols:
    raise RuntimeError(
        "Frozen endpoint policy-table columns changed."
    )


if policy_pre.shape != (
    100,
    10,
):
    raise RuntimeError(
        f"Frozen endpoint policy-table shape changed: "
        f"{policy_pre.shape}"
    )


if not np.array_equal(
    policy_pre[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        101,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Frozen endpoint unit ordering is not exactly 1..100; "
        "official RUL row alignment cannot be guaranteed."
    )


# --------------------------------------------------------------------------------------------
# 1) Verify ALL PRE-LABEL fingerprints BEFORE label access
# --------------------------------------------------------------------------------------------

def sha256_array_628hc6a9a4(
    x
):
    arr = np.ascontiguousarray(
        np.asarray(
            x
        )
    )

    return hashlib.sha256(
        arr.tobytes(
            order="C"
        )
    ).hexdigest()


fingerprints_expected = dict(
    FD001_TEST_PRELABEL_FINGERPRINTS_628HC6A9A3R1
)


fingerprints_now = {

    "endpoint_indices":
        sha256_array_628hc6a9a4(
            np.asarray(
                FD001_TEST_ENDPOINT_INDICES_628HC6A9A3R1,
                dtype=np.int64,
            )
        ),

    "deterministic_endpoint":
        sha256_array_628hc6a9a4(
            policy_pre[
                "deterministic_prediction"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "mc_mean_endpoint":
        sha256_array_628hc6a9a4(
            policy_pre[
                "mc_mean"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "mc_std_endpoint":
        sha256_array_628hc6a9a4(
            policy_pre[
                "mc_std"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "risk_endpoint":
        sha256_array_628hc6a9a4(
            policy_pre[
                "reliability_risk"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "accept_endpoint":
        sha256_array_628hc6a9a4(
            policy_pre[
                "accepted"
            ].to_numpy(
                dtype=np.uint8
            )
        ),
}


if set(
    fingerprints_now
) != set(
    fingerprints_expected
):
    raise RuntimeError(
        "Pre-label fingerprint key set changed."
    )


fingerprint_rows = []

for key in fingerprints_expected:

    same = bool(
        fingerprints_now[
            key
        ]
        ==
        fingerprints_expected[
            key
        ]
    )

    fingerprint_rows.append(
        {
            "object":
                key,

            "unchanged":
                same,
        }
    )


FD001_TEST_PRELABEL_FINGERPRINT_AUDIT_628HC6A9A4 = pd.DataFrame(
    fingerprint_rows
)


if not bool(
    FD001_TEST_PRELABEL_FINGERPRINT_AUDIT_628HC6A9A4[
        "unchanged"
    ].all()
):
    raise RuntimeError(
        "A pre-label TEST fingerprint changed. "
        "Official RUL labels will NOT be read."
    )


threshold = float(
    FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
)


nominal_coverage = float(
    FD001_NOMINAL_COVERAGE_628HC6A7A2A2
)


risk_pre = (
    policy_pre[
        "reliability_risk"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


accept_pre = (
    policy_pre[
        "accepted"
    ]
    .to_numpy(
        dtype=bool
    )
)


abstain_pre = (
    policy_pre[
        "abstained"
    ]
    .to_numpy(
        dtype=bool
    )
)


if not np.array_equal(
    accept_pre,
    risk_pre
    <=
    threshold,
):
    raise RuntimeError(
        "Frozen accept decisions no longer match "
        "the frozen threshold comparator."
    )


if not np.array_equal(
    abstain_pre,
    ~accept_pre,
):
    raise RuntimeError(
        "Frozen abstain decisions are not "
        "the complement of accept decisions."
    )


print(
    "\n--- PRE-LABEL FREEZE VERIFICATION ---"
)

print(
    FD001_TEST_PRELABEL_FINGERPRINT_AUDIT_628HC6A9A4
    .to_string(
        index=False
    )
)

print(
    "All pre-label fingerprints unchanged :",
    True,
)

print(
    "Frozen threshold                     :",
    threshold,
)

print(
    "Frozen endpoint coverage             :",
    float(
        accept_pre.mean()
    ),
)

print(
    "Official RUL contents read so far    :",
    False,
)


# --------------------------------------------------------------------------------------------
# 2) Resolve the already-known official RUL path WITHOUT opening it
#
# Multiple notebook variables may point to the same path.
# We deduplicate by resolved filesystem path.
# --------------------------------------------------------------------------------------------

rul_path_candidates = []


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        (
            str,
            Path,
        ),
    ):
        continue

    try:
        p = Path(
            obj
        )
    except Exception:
        continue

    try:

        if (
            p.name.lower()
            ==
            "rul_fd001.txt"

            and

            p.is_file()
        ):

            rul_path_candidates.append(
                p.resolve()
            )

    except (
        OSError,
        ValueError,
    ):
        continue


unique_rul_paths = sorted(
    set(
        rul_path_candidates
    ),
    key=lambda p:
        str(
            p
        ).lower(),
)


if len(
    unique_rul_paths
) != 1:
    raise RuntimeError(
        "Expected exactly one unique existing "
        "RUL_FD001.txt path already locked in memory; "
        f"found {len(unique_rul_paths)}: "
        f"{[str(p) for p in unique_rul_paths]}"
    )


rul_path = (
    unique_rul_paths[
        0
    ]
)


print(
    "\n--- OFFICIAL LABEL SOURCE LOCK ---"
)

print(
    "Official RUL path :",
    rul_path,
)

print(
    "Unique path count :",
    len(
        unique_rul_paths
    ),
)


# --------------------------------------------------------------------------------------------
# 3) ONE-TIME official RUL read
#
# If a partial execution somehow already performed the read,
# reuse the in-memory label vector instead of opening the file again.
# --------------------------------------------------------------------------------------------

already_read = bool(
    globals().get(
        "FD001_OFFICIAL_RUL_READ_ONCE_628HC6A9A4",
        False,
    )
)


if already_read:

    if (
        "FD001_OFFICIAL_RUL_628HC6A9A4"
        not in globals()
    ):
        raise RuntimeError(
            "Official-RUL read flag is set but "
            "the locked in-memory label vector is missing."
        )

    y_true = np.asarray(
        FD001_OFFICIAL_RUL_628HC6A9A4,
        dtype=np.float64,
    ).reshape(
        -1
    )

    rul_file_opened_this_execution = False


else:

    rul_df = pd.read_csv(
        rul_path,
        sep=r"\s+",
        header=None,
        engine="python",
    )


    if rul_df.shape != (
        100,
        1,
    ):
        raise RuntimeError(
            f"Official RUL contract mismatch: "
            f"{rul_df.shape}; expected (100, 1)."
        )


    y_true = (
        rul_df
        .iloc[
            :,
            0,
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    if not np.isfinite(
        y_true
    ).all():
        raise RuntimeError(
            "Official RUL labels contain non-finite values."
        )


    if bool(
        (
            y_true
            <
            0.0
        ).any()
    ):
        raise RuntimeError(
            "Official RUL labels contain negative values."
        )


    FD001_OFFICIAL_RUL_628HC6A9A4 = (
        y_true.copy()
    )


    FD001_OFFICIAL_RUL_READ_ONCE_628HC6A9A4 = True


    rul_file_opened_this_execution = True


if y_true.shape != (
    100,
):
    raise RuntimeError(
        f"Official RUL vector shape mismatch: "
        f"{y_true.shape}"
    )


# --------------------------------------------------------------------------------------------
# 4) FINAL scoring
#
# No prediction/risk/decision is recomputed here.
# We score only what was frozen BEFORE labels.
# --------------------------------------------------------------------------------------------

y_pred = (
    policy_pre[
        "deterministic_prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


if not np.isfinite(
    y_pred
).all():
    raise RuntimeError(
        "Frozen endpoint predictions contain non-finite values."
    )


abs_error = np.abs(
    y_pred
    -
    y_true
)


full_mae = float(
    np.mean(
        abs_error
    )
)


accepted_count = int(
    accept_pre.sum()
)


abstained_count = int(
    abstain_pre.sum()
)


if accepted_count <= 0:
    raise RuntimeError(
        "Frozen TEST policy accepted zero endpoints."
    )


if abstained_count <= 0:
    raise RuntimeError(
        "Frozen TEST policy abstained on zero endpoints."
    )


accepted_mae = float(
    np.mean(
        abs_error[
            accept_pre
        ]
    )
)


abstained_mae = float(
    np.mean(
        abs_error[
            abstain_pre
        ]
    )
)


selective_gain_mae = float(
    full_mae
    -
    accepted_mae
)


error_separation_mae = float(
    abstained_mae
    -
    accepted_mae
)


realized_coverage = float(
    accept_pre.mean()
)


coverage_delta_vs_nominal = float(
    realized_coverage
    -
    nominal_coverage
)


# --------------------------------------------------------------------------------------------
# 5) Freeze final official TEST result table in memory
# --------------------------------------------------------------------------------------------

FD001_TEST_FINAL_RESULTS_628HC6A9A4 = (
    policy_pre.copy(
        deep=True
    )
)


FD001_TEST_FINAL_RESULTS_628HC6A9A4[
    "official_rul"
] = y_true


FD001_TEST_FINAL_RESULTS_628HC6A9A4[
    "abs_error"
] = abs_error


FD001_TEST_FINAL_METRICS_628HC6A9A4 = pd.DataFrame(
    [
        {
            "endpoints":
                100,

            "accepted":
                accepted_count,

            "abstained":
                abstained_count,

            "coverage":
                realized_coverage,

            "coverage_delta_vs_nominal":
                coverage_delta_vs_nominal,

            "full_mae":
                full_mae,

            "accepted_mae":
                accepted_mae,

            "abstained_mae":
                abstained_mae,

            "selective_gain_mae":
                selective_gain_mae,

            "abstained_minus_accepted_mae":
                error_separation_mae,
        }
    ]
)


# --------------------------------------------------------------------------------------------
# 6) POST-LABEL integrity check
#
# The PRE-LABEL prediction/risk/decision fingerprints must still be identical.
# --------------------------------------------------------------------------------------------

fingerprints_after = {

    "endpoint_indices":
        sha256_array_628hc6a9a4(
            np.asarray(
                FD001_TEST_ENDPOINT_INDICES_628HC6A9A3R1,
                dtype=np.int64,
            )
        ),

    "deterministic_endpoint":
        sha256_array_628hc6a9a4(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
                "deterministic_prediction"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "mc_mean_endpoint":
        sha256_array_628hc6a9a4(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
                "mc_mean"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "mc_std_endpoint":
        sha256_array_628hc6a9a4(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
                "mc_std"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "risk_endpoint":
        sha256_array_628hc6a9a4(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
                "reliability_risk"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "accept_endpoint":
        sha256_array_628hc6a9a4(
            FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
                "accepted"
            ].to_numpy(
                dtype=np.uint8
            )
        ),
}


postlabel_fingerprints_unchanged = bool(
    fingerprints_after
    ==
    fingerprints_expected
)


threshold_unchanged = bool(
    float(
        FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
    )
    ==
    threshold
)


decisions_unchanged = bool(

    np.array_equal(
        FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
            "accepted"
        ].to_numpy(
            dtype=bool
        ),
        accept_pre,
    )

    and

    np.array_equal(
        FD001_TEST_ENDPOINT_POLICY_628HC6A9A3R1[
            "abstained"
        ].to_numpy(
            dtype=bool
        ),
        abstain_pre,
    )
)


# --------------------------------------------------------------------------------------------
# 7) Final audit
# --------------------------------------------------------------------------------------------

audit_rows = [

    (
        "prelabel_policy_frozen",
        True,
    ),

    (
        "prelabel_fingerprints_verified_before_label_read",
        True,
    ),

    (
        "official_rul_unique_path_1",
        len(
            unique_rul_paths
        )
        ==
        1,
    ),

    (
        "official_rul_shape_100",
        y_true.shape
        ==
        (
            100,
        ),
    ),

    (
        "official_rul_finite",
        bool(
            np.isfinite(
                y_true
            ).all()
        ),
    ),

    (
        "endpoint_unit_order_exact_1_to_100",
        True,
    ),

    (
        "prediction_shape_100",
        y_pred.shape
        ==
        (
            100,
        ),
    ),

    (
        "accepted_plus_abstained_100",
        accepted_count
        +
        abstained_count
        ==
        100,
    ),

    (
        "accepted_decisions_not_recomputed_from_labels",
        True,
    ),

    (
        "postlabel_prelabel_fingerprints_unchanged",
        postlabel_fingerprints_unchanged,
    ),

    (
        "threshold_unchanged",
        threshold_unchanged,
    ),

    (
        "decisions_unchanged_after_label_read",
        decisions_unchanged,
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_mc_dropout_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_or_calibration_tuning",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]


FD001_TEST_FINAL_AUDIT_628HC6A9A4 = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)


all_checks_passed = bool(
    FD001_TEST_FINAL_AUDIT_628HC6A9A4[
        "passed"
    ].all()
)


if not all_checks_passed:
    raise RuntimeError(
        "C6-A9-A4 final audit failed. "
        "Do not alter the frozen policy."
    )


FD001_TEST_FINAL_SCORING_FROZEN_628HC6A9A4 = True


# --------------------------------------------------------------------------------------------
# 8) Reporting
# --------------------------------------------------------------------------------------------

print(
    "\n--- FINAL OFFICIAL FD001 TEST METRICS ---"
)

print(
    FD001_TEST_FINAL_METRICS_628HC6A9A4
    .to_string(
        index=False
    )
)


print(
    "\n--- ACCEPTED / ABSTAINED ERROR CHECK ---"
)

print(
    "Full TEST MAE                     :",
    full_mae,
)

print(
    "Accepted TEST MAE                 :",
    accepted_mae,
)

print(
    "Abstained TEST MAE                :",
    abstained_mae,
)

print(
    "Selective gain in MAE             :",
    selective_gain_mae,
)

print(
    "Abstained - accepted MAE          :",
    error_separation_mae,
)

print(
    "Frozen endpoint coverage          :",
    realized_coverage,
)

print(
    "Coverage delta vs nominal 0.80    :",
    coverage_delta_vs_nominal,
)


print(
    "\n--- C6-A9-A4 AUDIT ---"
)

print(
    FD001_TEST_FINAL_AUDIT_628HC6A9A4
    .to_string(
        index=False
    )
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "STEP 6.28H-C6-A9-A4 FINAL SUMMARY"
)

print(
    "=" * 120
)


print(
    "Official RUL file opened this execution :",
    rul_file_opened_this_execution,
)

print(
    "Official RUL vector locked in memory    :",
    True,
)

print(
    "Pre-label fingerprints unchanged        :",
    postlabel_fingerprints_unchanged,
)

print(
    "Frozen threshold unchanged              :",
    threshold_unchanged,
)

print(
    "Frozen decisions unchanged              :",
    decisions_unchanged,
)

print(
    "Accepted / abstained endpoints          :",
    f"{accepted_count} / {abstained_count}",
)

print(
    "Realized TEST coverage                  :",
    realized_coverage,
)

print(
    "Full TEST MAE                           :",
    full_mae,
)

print(
    "Accepted TEST MAE                       :",
    accepted_mae,
)

print(
    "Abstained TEST MAE                      :",
    abstained_mae,
)

print(
    "Selective MAE gain                      :",
    selective_gain_mae,
)

print(
    "Model executed in this step             :",
    False,
)

print(
    "MC dropout executed in this step        :",
    False,
)

print(
    "Model retrained                         :",
    False,
)

print(
    "Threshold/calibration tuned             :",
    False,
)

print(
    "Files written                           :",
    False,
)

print(
    "All C6-A9-A4 checks passed              :",
    all_checks_passed,
)


print(
    "\nPASS — official FD001 TEST scoring was performed "
    "only AFTER predictions, uncertainty, risk, and "
    "accept/abstain decisions were frozen."
)

print(
    "The TEST result is now evaluation-only. "
    "Do not use it to retune the policy."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A4 ==="
)

In [ ]:
# ============================================================================================
# STEP 6.28H-C6-A9-A5 — FD001 FINAL TEST EVIDENCE LOCK / REPORTING-ONLY SUMMARY
#
# Purpose:
#   Freeze the already-scored TEST evidence in memory and produce final reporting metrics.
#
# No model execution.
# No MC dropout.
# No retraining.
# No threshold/calibration tuning.
# No policy modification.
# No file writing.
# ============================================================================================

import hashlib
import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 6.28H-C6-A9-A5 — FD001 FINAL TEST EVIDENCE LOCK / REPORTING-ONLY SUMMARY")
print("=" * 120)


# --------------------------------------------------------------------------------------------
# 0) Required final objects
# --------------------------------------------------------------------------------------------

required = [
    "FD001_TEST_FINAL_METRICS_628HC6A9A4",
    "FD001_TEST_FINAL_RESULTS_628HC6A9A4",
    "FD001_TEST_FINAL_AUDIT_628HC6A9A4",
    "FD001_TEST_FINAL_SCORING_FROZEN_628HC6A9A4",
    "FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required final TEST object(s): "
        + ", ".join(missing)
    )


if not bool(
    FD001_TEST_FINAL_SCORING_FROZEN_628HC6A9A4
):
    raise RuntimeError(
        "Final TEST scoring is not frozen. Stop."
    )


metrics = (
    FD001_TEST_FINAL_METRICS_628HC6A9A4
    .copy(
        deep=True
    )
)

results = (
    FD001_TEST_FINAL_RESULTS_628HC6A9A4
    .copy(
        deep=True
    )
)

audit_prev = (
    FD001_TEST_FINAL_AUDIT_628HC6A9A4
    .copy(
        deep=True
    )
)


if metrics.shape[0] != 1:
    raise RuntimeError(
        f"Expected one final-metrics row, found {metrics.shape[0]}."
    )


if results.shape[0] != 100:
    raise RuntimeError(
        f"Expected 100 final endpoint rows, found {results.shape[0]}."
    )


if not bool(
    audit_prev["passed"].all()
):
    raise RuntimeError(
        "Previous final TEST audit is not fully passing."
    )


# --------------------------------------------------------------------------------------------
# 1) Recover frozen final metrics only
# --------------------------------------------------------------------------------------------

row = metrics.iloc[0]


endpoints = int(
    row["endpoints"]
)

accepted = int(
    row["accepted"]
)

abstained = int(
    row["abstained"]
)

coverage = float(
    row["coverage"]
)

coverage_delta = float(
    row["coverage_delta_vs_nominal"]
)

full_mae = float(
    row["full_mae"]
)

accepted_mae = float(
    row["accepted_mae"]
)

abstained_mae = float(
    row["abstained_mae"]
)

selective_gain = float(
    row["selective_gain_mae"]
)

error_separation = float(
    row["abstained_minus_accepted_mae"]
)


relative_mae_reduction_pct = float(
    100.0
    *
    selective_gain
    /
    full_mae
)


abstained_vs_accepted_increase_pct = float(
    100.0
    *
    error_separation
    /
    accepted_mae
)


# --------------------------------------------------------------------------------------------
# 2) Final evidence fingerprints
# --------------------------------------------------------------------------------------------

def sha256_numeric_628hc6a9a5(
    x
):
    arr = np.ascontiguousarray(
        np.asarray(
            x
        )
    )

    return hashlib.sha256(
        arr.tobytes(
            order="C"
        )
    ).hexdigest()


final_fingerprints = {

    "official_rul":
        sha256_numeric_628hc6a9a5(
            results[
                "official_rul"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "deterministic_prediction":
        sha256_numeric_628hc6a9a5(
            results[
                "deterministic_prediction"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "mc_std":
        sha256_numeric_628hc6a9a5(
            results[
                "mc_std"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "reliability_risk":
        sha256_numeric_628hc6a9a5(
            results[
                "reliability_risk"
            ].to_numpy(
                dtype=np.float64
            )
        ),

    "accepted":
        sha256_numeric_628hc6a9a5(
            results[
                "accepted"
            ].to_numpy(
                dtype=np.uint8
            )
        ),

    "abs_error":
        sha256_numeric_628hc6a9a5(
            results[
                "abs_error"
            ].to_numpy(
                dtype=np.float64
            )
        ),
}


FD001_TEST_FINAL_EVIDENCE_FINGERPRINTS_628HC6A9A5 = dict(
    final_fingerprints
)


# --------------------------------------------------------------------------------------------
# 3) Reporting-only final summary
# --------------------------------------------------------------------------------------------

FD001_TEST_FINAL_REPORT_628HC6A9A5 = pd.DataFrame(
    [
        {
            "endpoints":
                endpoints,

            "accepted":
                accepted,

            "abstained":
                abstained,

            "coverage":
                coverage,

            "coverage_delta_vs_nominal":
                coverage_delta,

            "full_mae":
                full_mae,

            "accepted_mae":
                accepted_mae,

            "abstained_mae":
                abstained_mae,

            "selective_gain_mae":
                selective_gain,

            "relative_mae_reduction_pct":
                relative_mae_reduction_pct,

            "abstained_minus_accepted_mae":
                error_separation,

            "abstained_vs_accepted_increase_pct":
                abstained_vs_accepted_increase_pct,
        }
    ]
)


# --------------------------------------------------------------------------------------------
# 4) Final immutable-evidence audit
# --------------------------------------------------------------------------------------------

audit_rows = [

    (
        "previous_final_scoring_frozen",
        True,
    ),

    (
        "previous_final_audit_passed",
        bool(
            audit_prev[
                "passed"
            ].all()
        ),
    ),

    (
        "final_results_rows_100",
        results.shape[0]
        ==
        100,
    ),

    (
        "accepted_plus_abstained_100",
        accepted
        +
        abstained
        ==
        100,
    ),

    (
        "coverage_matches_counts",
        np.isclose(
            coverage,
            accepted
            /
            endpoints,
        ),
    ),

    (
        "accepted_mae_below_full_mae",
        accepted_mae
        <
        full_mae,
    ),

    (
        "abstained_mae_above_full_mae",
        abstained_mae
        >
        full_mae,
    ),

    (
        "abstained_mae_above_accepted_mae",
        abstained_mae
        >
        accepted_mae,
    ),

    (
        "selective_gain_positive",
        selective_gain
        >
        0.0,
    ),

    (
        "frozen_threshold_unchanged",
        np.isclose(
            float(
                FD001_FROZEN_RISK_THRESHOLD_628HC6A7A2A2
            ),
            0.7983391809203397,
            rtol=0.0,
            atol=0.0,
        ),
    ),

    (
        "no_model_execution",
        True,
    ),

    (
        "no_mc_dropout_execution",
        True,
    ),

    (
        "no_model_retraining",
        True,
    ),

    (
        "no_threshold_or_calibration_tuning",
        True,
    ),

    (
        "no_policy_modification",
        True,
    ),

    (
        "no_files_written",
        True,
    ),
]


FD001_TEST_FINAL_EVIDENCE_AUDIT_628HC6A9A5 = pd.DataFrame(
    audit_rows,
    columns=[
        "check",
        "passed",
    ],
)


all_checks_passed = bool(
    FD001_TEST_FINAL_EVIDENCE_AUDIT_628HC6A9A5[
        "passed"
    ].all()
)


if not all_checks_passed:
    raise RuntimeError(
        "C6-A9-A5 final evidence-lock audit failed."
    )


FD001_TEST_FINAL_EVIDENCE_LOCKED_628HC6A9A5 = True


# --------------------------------------------------------------------------------------------
# 5) Reporting
# --------------------------------------------------------------------------------------------

print(
    "\n--- FINAL REPORTING METRICS ---"
)

print(
    FD001_TEST_FINAL_REPORT_628HC6A9A5
    .to_string(
        index=False
    )
)


print(
    "\n--- FINAL EVIDENCE FINGERPRINTS ---"
)

print(
    pd.DataFrame(
        [
            {
                "object":
                    k,

                "sha256":
                    v,
            }
            for k, v
            in final_fingerprints.items()
        ]
    )
    .to_string(
        index=False
    )
)


print(
    "\n--- C6-A9-A5 AUDIT ---"
)

print(
    FD001_TEST_FINAL_EVIDENCE_AUDIT_628HC6A9A5
    .to_string(
        index=False
    )
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "STEP 6.28H-C6-A9-A5 FINAL SUMMARY"
)

print(
    "=" * 120
)


print(
    "Official TEST endpoints                  :",
    endpoints,
)

print(
    "Accepted / abstained                     :",
    f"{accepted} / {abstained}",
)

print(
    "Realized coverage                        :",
    coverage,
)

print(
    "Nominal coverage gap                     :",
    coverage_delta,
)

print(
    "Full TEST MAE                            :",
    full_mae,
)

print(
    "Accepted TEST MAE                        :",
    accepted_mae,
)

print(
    "Abstained TEST MAE                       :",
    abstained_mae,
)

print(
    "Selective MAE gain                       :",
    selective_gain,
)

print(
    "Relative MAE reduction (%)               :",
    relative_mae_reduction_pct,
)

print(
    "Abstained - accepted MAE                 :",
    error_separation,
)

print(
    "Abstained vs accepted error increase (%) :",
    abstained_vs_accepted_increase_pct,
)

print(
    "Final TEST evidence locked               :",
    True,
)

print(
    "Model executed                           :",
    False,
)

print(
    "MC dropout executed                      :",
    False,
)

print(
    "Model retrained                          :",
    False,
)

print(
    "Threshold/calibration tuned              :",
    False,
)

print(
    "Policy modified                          :",
    False,
)

print(
    "Files written                            :",
    False,
)

print(
    "All C6-A9-A5 checks passed               :",
    all_checks_passed,
)


print(
    "\nPASS — FD001 TEST evidence is now reporting-only. "
    "Do not use these TEST outcomes to retune the frozen policy."
)

print(
    "\n=== END STEP 6.28H-C6-A9-A5 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A0 — FD002 CURRENT FROZEN-STATE / UNFINISHED-SELECTION RECOVERY
# Read-only forensic inventory. No model execution, no MC dropout, no TEST scoring, no file writes.
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C7-A0 — FD002 CURRENT FROZEN-STATE / UNFINISHED-SELECTION RECOVERY")
print("=" * 118)

# Stable snapshot: avoids "dictionary changed size during iteration".
G_628hc7a0 = dict(list(globals().items()))

# --------------------------------------------------------------------------------------
# 1) Exact FD002 objects expected from the completed C2/C3 development chain
# --------------------------------------------------------------------------------------
critical_names_628hc7a0 = [
    "uncertainty_risk_matrix_628hc3a0r1",
    "clean_uncertainty_risk_628hc3a0r1",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "CLEAN_DEV_SENSOR_QUALITY_CALIBRATED_628HC2B",
    "FD002_LEVEL_RISK_MATRIX_628HC3A3A2",
    "FD002_NOISE_RISK_MATRIX_628HC3A3A2",
    "FD002_DROPOUT_RISK_MATRIX_628HC3A3A2",
    "FD002_STUCK_RISK_MATRIX_628HC3A3A2",
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
]

def summarize_object_628hc7a0(name):
    exists = name in G_628hc7a0

    if not exists:
        return {
            "name": name,
            "exists": False,
            "type": None,
            "shape_or_len": None,
            "finite": None,
            "in_0_1": None,
        }

    obj = G_628hc7a0[name]
    obj_type = type(obj).__name__

    if isinstance(obj, np.ndarray):
        shape_or_len = str(obj.shape)

        if np.issubdtype(obj.dtype, np.number):
            finite = bool(np.isfinite(obj).all())
            in_0_1 = bool(
                finite
                and np.nanmin(obj) >= 0.0
                and np.nanmax(obj) <= 1.0
            )
        else:
            finite = None
            in_0_1 = None

    elif isinstance(obj, pd.DataFrame):
        shape_or_len = str(obj.shape)
        finite = None
        in_0_1 = None

    elif isinstance(obj, dict):
        shape_or_len = f"dict[{len(obj)}]"
        finite = None
        in_0_1 = None

    elif hasattr(obj, "__len__") and not isinstance(obj, (str, bytes)):
        try:
            shape_or_len = f"len={len(obj)}"
        except Exception:
            shape_or_len = None

        finite = None
        in_0_1 = None

    else:
        shape_or_len = "scalar"
        finite = None
        in_0_1 = None

    return {
        "name": name,
        "exists": True,
        "type": obj_type,
        "shape_or_len": shape_or_len,
        "finite": finite,
        "in_0_1": in_0_1,
    }


critical_table_628hc7a0 = pd.DataFrame(
    [
        summarize_object_628hc7a0(name)
        for name in critical_names_628hc7a0
    ]
)

print("\n--- CRITICAL FD002 DEVELOPMENT OBJECTS ---")
display(critical_table_628hc7a0)

# --------------------------------------------------------------------------------------
# 2) Contract checks for the exact unfinished FD002 reliability-selection state
# --------------------------------------------------------------------------------------
EXPECTED_SCENARIOS_628hc7a0 = 315
EXPECTED_WINDOWS_628hc7a0 = 8367

EXPECTED_MATRIX_SHAPE_628hc7a0 = (
    EXPECTED_SCENARIOS_628hc7a0,
    EXPECTED_WINDOWS_628hc7a0,
)

matrix_names_628hc7a0 = [
    "uncertainty_risk_matrix_628hc3a0r1",
    "FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C",
    "FD002_LEVEL_RISK_MATRIX_628HC3A3A2",
    "FD002_NOISE_RISK_MATRIX_628HC3A3A2",
    "FD002_DROPOUT_RISK_MATRIX_628HC3A3A2",
    "FD002_STUCK_RISK_MATRIX_628HC3A3A2",
]

matrix_contract_rows_628hc7a0 = []

for name in matrix_names_628hc7a0:

    obj = G_628hc7a0.get(
        name,
        None,
    )

    shape_ok = (
        isinstance(obj, np.ndarray)
        and obj.shape == EXPECTED_MATRIX_SHAPE_628hc7a0
    )

    finite_ok = (
        shape_ok
        and bool(np.isfinite(obj).all())
    )

    range_ok = (
        finite_ok
        and bool(np.min(obj) >= 0.0)
        and bool(np.max(obj) <= 1.0)
    )

    matrix_contract_rows_628hc7a0.append(
        {
            "name": name,
            "shape_315x8367": bool(shape_ok),
            "finite": bool(finite_ok),
            "in_0_1": bool(range_ok),
        }
    )


matrix_contract_628hc7a0 = pd.DataFrame(
    matrix_contract_rows_628hc7a0
)

print("\n--- MATRIX CONTRACT ---")
display(matrix_contract_628hc7a0)

clean_u_628hc7a0 = G_628hc7a0.get(
    "clean_uncertainty_risk_628hc3a0r1",
    None,
)

clean_u_ok_628hc7a0 = (
    isinstance(clean_u_628hc7a0, np.ndarray)
    and clean_u_628hc7a0.shape == (EXPECTED_WINDOWS_628hc7a0,)
    and np.isfinite(clean_u_628hc7a0).all()
    and np.min(clean_u_628hc7a0) >= 0.0
    and np.max(clean_u_628hc7a0) <= 1.0
)

response_628hc7a0 = G_628hc7a0.get(
    "FD002_SENSOR_QUALITY_RESPONSE_628HC2C",
    None,
)

response_ok_628hc7a0 = (
    isinstance(response_628hc7a0, pd.DataFrame)
    and len(response_628hc7a0) == EXPECTED_SCENARIOS_628hc7a0
)

all_matrix_contracts_ok_628hc7a0 = bool(
    matrix_contract_628hc7a0[
        [
            "shape_315x8367",
            "finite",
            "in_0_1",
        ]
    ]
    .to_numpy(dtype=bool)
    .all()
)

# --------------------------------------------------------------------------------------
# 3) Read-only inventory of FD002 selection/freeze-related names
# --------------------------------------------------------------------------------------
selection_keywords_628hc7a0 = (
    "FROZEN",
    "THRESH",
    "FINAL_RISK",
    "COMBINED",
    "FUSION",
    "ADAPT",
    "GATE",
    "GAMMA",
    "C3A3",
    "C3_A3",
)

selection_names_628hc7a0 = sorted(
    name
    for name in G_628hc7a0
    if "FD002" in name.upper()
    and any(
        key in name.upper()
        for key in selection_keywords_628hc7a0
    )
)

selection_table_628hc7a0 = pd.DataFrame(
    {
        "name": selection_names_628hc7a0,
        "type": [
            type(G_628hc7a0[name]).__name__
            for name in selection_names_628hc7a0
        ],
    }
)

print("\n--- FD002 SELECTION / FREEZE-RELATED OBJECT NAMES ---")

if len(selection_table_628hc7a0) == 0:
    print(
        "No matching FD002 selection/freeze object names found."
    )
else:
    display(selection_table_628hc7a0)

# --------------------------------------------------------------------------------------
# 4) Audit
# --------------------------------------------------------------------------------------
audit_628hc7a0 = pd.DataFrame(
    {
        "check": [
            "six_required_risk_matrices_valid",
            "clean_uncertainty_risk_valid",
            "sensor_quality_response_has_315_rows",
            "no_model_execution",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_threshold_selection_or_modification",
            "no_test_scoring",
            "no_files_written",
        ],
        "passed": [
            all_matrix_contracts_ok_628hc7a0,
            bool(clean_u_ok_628hc7a0),
            bool(response_ok_628hc7a0),
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C7-A0 AUDIT ---")
display(audit_628hc7a0)

all_checks_passed_628hc7a0 = bool(
    audit_628hc7a0["passed"].all()
)

print("\n" + "=" * 118)
print("STEP 6.28H-C7-A0 FINAL SUMMARY")
print("=" * 118)

print(
    "FD002 uncertainty + quality branch state recovered :",
    all_matrix_contracts_ok_628hc7a0,
)

print(
    "Clean uncertainty reference recovered             :",
    bool(clean_u_ok_628hc7a0),
)

print(
    "315-scenario quality response recovered           :",
    bool(response_ok_628hc7a0),
)

print(
    "Selection/freeze-related names found              :",
    len(selection_names_628hc7a0),
)

print(
    "Model executed                                    : False"
)

print(
    "MC dropout executed                               : False"
)

print(
    "TEST scored                                       : False"
)

print(
    "Threshold/rule modified                           : False"
)

print(
    "Files written                                     : False"
)

print(
    "All C7-A0 contract checks passed                  :",
    all_checks_passed_628hc7a0,
)

if not all_checks_passed_628hc7a0:

    raise RuntimeError(
        "C7-A0 recovery contract failed. "
        "Do not continue to FD002 adaptive-rule selection."
    )

print(
    "\nPASS — FD002 DEV reliability state is recoverable "
    "without rerunning expensive experiments."
)

print(
    "Next: C7-A1 — recover/lock the exact FD002 unfinished "
    "adaptive reliability-selection point using DEV only; "
    "TEST remains untouched."
)

print(
    "\n=== END STEP 6.28H-C7-A0 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A1 — FD002 UNFINISHED ADAPTIVE-SELECTION POINT RECOVERY
#
# PURPOSE
# -------
# Recover the exact historical C3-A2 / C3-A3 diagnostic state and locate the existing
# DEV-only prediction/error/target objects needed for adaptive reliability evaluation.
#
# IMPORTANT
# ---------
# READ-ONLY.
# No model execution.
# No MC dropout.
# No retraining.
# No TEST access/scoring.
# No threshold/rule selection yet.
# No files written.
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 118)
print("STEP 6.28H-C7-A1 — FD002 UNFINISHED ADAPTIVE-SELECTION POINT RECOVERY")
print("=" * 118)

# Stable globals snapshot.
G_628hc7a1 = dict(list(globals().items()))

# --------------------------------------------------------------------------------------
# 1) Lock the already-recovered matrices from C7-A0
# --------------------------------------------------------------------------------------

U_628hc7a1 = np.asarray(
    G_628hc7a1["uncertainty_risk_matrix_628hc3a0r1"],
    dtype=np.float64,
)

Q_628hc7a1 = np.asarray(
    G_628hc7a1["FD002_SENSOR_QUALITY_RISK_MATRIX_628HC2C"],
    dtype=np.float64,
)

LEVEL_628hc7a1 = np.asarray(
    G_628hc7a1["FD002_LEVEL_RISK_MATRIX_628HC3A3A2"],
    dtype=np.float64,
)

NOISE_628hc7a1 = np.asarray(
    G_628hc7a1["FD002_NOISE_RISK_MATRIX_628HC3A3A2"],
    dtype=np.float64,
)

DROPOUT_628hc7a1 = np.asarray(
    G_628hc7a1["FD002_DROPOUT_RISK_MATRIX_628HC3A3A2"],
    dtype=np.float64,
)

STUCK_628hc7a1 = np.asarray(
    G_628hc7a1["FD002_STUCK_RISK_MATRIX_628HC3A3A2"],
    dtype=np.float64,
)

EXPECTED_SHAPE_628hc7a1 = (
    315,
    8367,
)

base_contract_ok_628hc7a1 = bool(
    U_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
    and Q_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
    and LEVEL_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
    and NOISE_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
    and DROPOUT_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
    and STUCK_628hc7a1.shape == EXPECTED_SHAPE_628hc7a1
)

# --------------------------------------------------------------------------------------
# 2) Exact historical C3-A2 / C3-A3 object inventory
# --------------------------------------------------------------------------------------

history_tokens_628hc7a1 = (
    "C3A2",
    "C3_A2",
    "C3A3",
    "C3_A3",
)

history_rows_628hc7a1 = []

for name, obj in G_628hc7a1.items():

    upper_name = name.upper()

    if "FD002" not in upper_name:
        continue

    if not any(
        token in upper_name
        for token in history_tokens_628hc7a1
    ):
        continue

    if isinstance(obj, pd.DataFrame):

        shape_or_len = str(
            obj.shape
        )

        columns_or_keys_preview = (
            list(obj.columns)[:12]
        )

    elif isinstance(obj, np.ndarray):

        shape_or_len = str(
            obj.shape
        )

        columns_or_keys_preview = None

    elif isinstance(obj, dict):

        shape_or_len = (
            f"dict[{len(obj)}]"
        )

        columns_or_keys_preview = (
            list(obj.keys())[:12]
        )

    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):

        shape_or_len = (
            f"len={len(obj)}"
        )

        columns_or_keys_preview = None

    else:

        shape_or_len = "scalar"

        columns_or_keys_preview = None

    history_rows_628hc7a1.append(
        {
            "name": name,
            "type": type(obj).__name__,
            "shape_or_len": shape_or_len,
            "columns_or_keys_preview": columns_or_keys_preview,
        }
    )


history_table_628hc7a1 = (
    pd.DataFrame(
        history_rows_628hc7a1
    )
    .sort_values(
        "name"
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- FD002 C3-A2 / C3-A3 HISTORICAL OBJECTS ---"
)

display(
    history_table_628hc7a1
)

# --------------------------------------------------------------------------------------
# 3) Recover the known fixed-gamma diagnostic objects
# --------------------------------------------------------------------------------------

known_c3a2_names_628hc7a1 = [
    "FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2",
    "FD002_C3A2_GLOBAL_GAMMA_SUMMARY_628HC3A2",
    "FD002_C3A2_GAMMA_STABILITY_628HC3A2",
    "FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2",
    "FD002_C3A2_CLEAN_GAMMA_RESPONSE_628HC3A2",
]

known_c3a2_rows_628hc7a1 = []

for name in known_c3a2_names_628hc7a1:

    obj = G_628hc7a1.get(
        name,
        None,
    )

    known_c3a2_rows_628hc7a1.append(
        {
            "name": name,
            "exists": obj is not None,
            "type": (
                None
                if obj is None
                else type(obj).__name__
            ),
            "shape": (
                None
                if obj is None
                else str(
                    getattr(
                        obj,
                        "shape",
                        None,
                    )
                )
            ),
        }
    )


known_c3a2_table_628hc7a1 = pd.DataFrame(
    known_c3a2_rows_628hc7a1
)

print(
    "\n--- KNOWN C3-A2 FIXED-GAMMA OBJECTS ---"
)

display(
    known_c3a2_table_628hc7a1
)

if isinstance(
    G_628hc7a1.get(
        "FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2",
        None,
    ),
    pd.DataFrame,
):

    print(
        "\n--- GLOBAL GAMMA RANKING: TOP 10 ---"
    )

    display(
        G_628hc7a1[
            "FD002_C3A2_GLOBAL_GAMMA_RANKING_628HC3A2"
        ].head(
            10
        )
    )


if isinstance(
    G_628hc7a1.get(
        "FD002_C3A2_GAMMA_STABILITY_628HC3A2",
        None,
    ),
    pd.DataFrame,
):

    print(
        "\n--- GAMMA STABILITY / LOFO TABLE ---"
    )

    display(
        G_628hc7a1[
            "FD002_C3A2_GAMMA_STABILITY_628HC3A2"
        ]
    )


if isinstance(
    G_628hc7a1.get(
        "FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2",
        None,
    ),
    pd.DataFrame,
):

    print(
        "\n--- GAMMA VS UNCERTAINTY-ONLY ---"
    )

    display(
        G_628hc7a1[
            "FD002_C3A2_GAMMA_VS_UNCERTAINTY_628HC3A2"
        ]
    )

# --------------------------------------------------------------------------------------
# 4) Locate existing DEV error / prediction / target objects
#
# We do NOT score anything here.
# We only identify the exact existing objects needed by C7-A2.
# --------------------------------------------------------------------------------------

candidate_tokens_628hc7a1 = (
    "ERROR",
    "ABS_ERR",
    "ABSERROR",
    "ABS_ERROR",
    "PRED",
    "TARGET",
    "Y_DEV",
    "DEV_Y",
    "RUL",
)

candidate_rows_628hc7a1 = []

for name, obj in G_628hc7a1.items():

    upper_name = name.upper()

    if not any(
        token in upper_name
        for token in candidate_tokens_628hc7a1
    ):
        continue

    if isinstance(
        obj,
        np.ndarray,
    ):

        if obj.shape not in {
            (8367,),
            (315, 8367),
            (315,),
        }:
            continue

        if np.issubdtype(
            obj.dtype,
            np.number,
        ):

            finite = bool(
                np.isfinite(
                    obj
                ).all()
            )

        else:

            finite = None

        candidate_rows_628hc7a1.append(
            {
                "name": name,
                "type": "ndarray",
                "shape": str(
                    obj.shape
                ),
                "dtype": str(
                    obj.dtype
                ),
                "finite": finite,
                "min": (
                    float(
                        np.min(
                            obj
                        )
                    )
                    if finite
                    else np.nan
                ),
                "median": (
                    float(
                        np.median(
                            obj
                        )
                    )
                    if finite
                    else np.nan
                ),
                "max": (
                    float(
                        np.max(
                            obj
                        )
                    )
                    if finite
                    else np.nan
                ),
            }
        )

    elif isinstance(
        obj,
        pd.DataFrame,
    ):

        if len(obj) not in {
            315,
            8367,
        }:
            continue

        candidate_rows_628hc7a1.append(
            {
                "name": name,
                "type": "DataFrame",
                "shape": str(
                    obj.shape
                ),
                "dtype": None,
                "finite": None,
                "min": np.nan,
                "median": np.nan,
                "max": np.nan,
            }
        )


if candidate_rows_628hc7a1:

    candidate_table_628hc7a1 = (
        pd.DataFrame(
            candidate_rows_628hc7a1
        )
        .sort_values(
            [
                "shape",
                "name",
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    candidate_table_628hc7a1 = pd.DataFrame(
        columns=[
            "name",
            "type",
            "shape",
            "dtype",
            "finite",
            "min",
            "median",
            "max",
        ]
    )


print(
    "\n--- DEV ERROR / PREDICTION / TARGET CANDIDATES ---"
)

display(
    candidate_table_628hc7a1
)

# --------------------------------------------------------------------------------------
# 5) Recover scenario metadata from the existing 315-row quality-response table
# --------------------------------------------------------------------------------------

response_628hc7a1 = (
    G_628hc7a1[
        "FD002_SENSOR_QUALITY_RESPONSE_628HC2C"
    ]
    .copy()
)

metadata_cols_628hc7a1 = [
    c
    for c in [
        "scenario_id",
        "scenario_number",
        "family",
        "sensor",
        "severity",
        "alpha",
        "direction",
        "requested_fraction",
        "seed",
    ]
    if c in response_628hc7a1.columns
]

print(
    "\n--- AVAILABLE SCENARIO METADATA COLUMNS ---"
)

print(
    metadata_cols_628hc7a1
)

if metadata_cols_628hc7a1:

    display(
        response_628hc7a1[
            metadata_cols_628hc7a1
        ].head(
            12
        )
    )


metadata_contract_ok_628hc7a1 = bool(
    len(
        response_628hc7a1
    )
    == 315
    and "family"
    in response_628hc7a1.columns
    and "severity"
    in response_628hc7a1.columns
)

# --------------------------------------------------------------------------------------
# 6) Audit
# --------------------------------------------------------------------------------------

audit_628hc7a1 = pd.DataFrame(
    {
        "check": [
            "previous_C7_A0_passed",
            "six_risk_matrices_shape_315x8367",
            "c3a2_c3a3_history_recovered",
            "scenario_metadata_315_rows_with_family_severity",
            "no_model_execution",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_test_access_or_scoring",
            "no_threshold_or_rule_selection",
            "no_files_written",
        ],
        "passed": [
            bool(
                G_628hc7a1.get(
                    "all_checks_passed_628hc7a0",
                    False,
                )
            ),
            base_contract_ok_628hc7a1,
            bool(
                len(
                    history_table_628hc7a1
                )
                > 0
            ),
            metadata_contract_ok_628hc7a1,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print(
    "\n--- C7-A1 AUDIT ---"
)

display(
    audit_628hc7a1
)

all_checks_passed_628hc7a1 = bool(
    audit_628hc7a1[
        "passed"
    ].all()
)

# --------------------------------------------------------------------------------------
# 7) Final summary
# --------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 118
)

print(
    "STEP 6.28H-C7-A1 FINAL SUMMARY"
)

print(
    "=" * 118
)

print(
    "Historical C3-A2/C3-A3 objects recovered          :",
    len(
        history_table_628hc7a1
    ),
)

print(
    "DEV error/prediction/target candidates found      :",
    len(
        candidate_table_628hc7a1
    ),
)

print(
    "Scenario metadata contract recovered              :",
    metadata_contract_ok_628hc7a1,
)

print(
    "Model executed                                    : False"
)

print(
    "MC dropout executed                               : False"
)

print(
    "TEST accessed/scored                              : False"
)

print(
    "Threshold/rule selected or modified               : False"
)

print(
    "Files written                                     : False"
)

print(
    "All C7-A1 contract checks passed                  :",
    all_checks_passed_628hc7a1,
)


if not all_checks_passed_628hc7a1:

    raise RuntimeError(
        "C7-A1 recovery contract failed. "
        "Do not begin adaptive-rule evaluation."
    )


print(
    "\nPASS — the unfinished FD002 adaptive-selection context "
    "has been recovered without rerunning expensive experiments."
)

print(
    "Next: C7-A2 — evaluate branch-pattern adaptive gating on DEV only, "
    "using the recovered frozen uncertainty/quality matrices and existing DEV errors. "
    "Family/severity labels remain diagnostics only and are never runtime gate inputs."
)

print(
    "\n=== END STEP 6.28H-C7-A1 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A2 — FD002 HISTORICAL ADAPTIVE DECISION + FROZEN-RISK RECOVERY LOCK
#
# PURPOSE
# -------
# Recover and verify the exact FD002 adaptive decision that was ALREADY made historically.
#
# THIS CELL DOES NOT:
#   - tune a gate,
#   - select a new architecture,
#   - modify a threshold,
#   - run the model,
#   - run MC dropout,
#   - retrain,
#   - access TEST,
#   - score TEST,
#   - write files.
#
# If the historical frozen objects are valid, they become the authoritative FD002
# pre-TEST reliability state for the continuation of this project.
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C7-A2 — FD002 HISTORICAL ADAPTIVE DECISION + FROZEN-RISK RECOVERY LOCK")
print("=" * 122)

# Stable namespace snapshot.
G_628hc7a2 = dict(globals())

EXPECTED_SCENARIOS_628hc7a2 = 315
EXPECTED_WINDOWS_628hc7a2 = 8367


# --------------------------------------------------------------------------------------------------
# 1) Require previous recovery step
# --------------------------------------------------------------------------------------------------

previous_ok_628hc7a2 = bool(
    G_628hc7a2.get(
        "all_checks_passed_628hc7a1",
        False,
    )
)

if not previous_ok_628hc7a2:
    raise RuntimeError(
        "C7-A1 must pass before recovering the historical FD002 freeze."
    )


# --------------------------------------------------------------------------------------------------
# 2) Recover historical C3-A3-A7 decision objects
# --------------------------------------------------------------------------------------------------

required_decision_names_628hc7a2 = [
    "FD002_C3A3A3A7_ADAPTIVE_DECISION",
    "FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED",
    "FD002_C3A3A3A7_DECISION",
    "FD002_C3A3A3A7_SELECTED_ARCHITECTURE",
]

decision_presence_rows_628hc7a2 = []

for name in required_decision_names_628hc7a2:

    obj = G_628hc7a2.get(name, None)

    decision_presence_rows_628hc7a2.append(
        {
            "name": name,
            "exists": obj is not None,
            "type": None if obj is None else type(obj).__name__,
            "shape_or_value": (
                None
                if obj is None
                else (
                    str(obj.shape)
                    if hasattr(obj, "shape")
                    else str(obj)
                )
            ),
        }
    )

decision_presence_628hc7a2 = pd.DataFrame(
    decision_presence_rows_628hc7a2
)

print("\n--- HISTORICAL C3-A3-A7 DECISION OBJECTS ---")
display(decision_presence_628hc7a2)

decision_objects_present_628hc7a2 = bool(
    decision_presence_628hc7a2["exists"].all()
)

if not decision_objects_present_628hc7a2:
    raise RuntimeError(
        "Historical C3-A3-A7 decision objects are incomplete. "
        "Do not perform a new selection."
    )


historical_decision_table_628hc7a2 = G_628hc7a2[
    "FD002_C3A3A3A7_ADAPTIVE_DECISION"
]

historical_adaptive_accepted_628hc7a2 = bool(
    G_628hc7a2[
        "FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED"
    ]
)

historical_decision_text_628hc7a2 = str(
    G_628hc7a2[
        "FD002_C3A3A3A7_DECISION"
    ]
)

historical_selected_architecture_628hc7a2 = str(
    G_628hc7a2[
        "FD002_C3A3A3A7_SELECTED_ARCHITECTURE"
    ]
)

print("\n--- HISTORICAL ADAPTIVE DECISION TABLE ---")
display(historical_decision_table_628hc7a2)

print("\n--- HISTORICAL DECISION SCALARS ---")
print(
    "Adaptive fusion accepted :",
    historical_adaptive_accepted_628hc7a2,
)
print(
    "Decision text            :",
    historical_decision_text_628hc7a2,
)
print(
    "Selected architecture    :",
    historical_selected_architecture_628hc7a2,
)


# --------------------------------------------------------------------------------------------------
# 3) Recover ALL historical frozen FD002 objects
# --------------------------------------------------------------------------------------------------

frozen_names_628hc7a2 = sorted(
    [
        name
        for name in G_628hc7a2
        if name.startswith("FD002_FROZEN_")
    ]
)

frozen_rows_628hc7a2 = []

for name in frozen_names_628hc7a2:

    obj = G_628hc7a2[name]

    if isinstance(obj, np.ndarray):
        descriptor = str(obj.shape)

    elif isinstance(obj, pd.DataFrame):
        descriptor = str(obj.shape)

    elif isinstance(obj, dict):
        descriptor = f"dict[{len(obj)}]"

    elif isinstance(obj, (list, tuple)):
        descriptor = f"len={len(obj)}"

    else:
        descriptor = str(obj)

    frozen_rows_628hc7a2.append(
        {
            "name": name,
            "type": type(obj).__name__,
            "shape_or_value": descriptor,
        }
    )

frozen_inventory_628hc7a2 = pd.DataFrame(
    frozen_rows_628hc7a2
)

print("\n--- ALL FD002_FROZEN_* OBJECTS ---")
display(frozen_inventory_628hc7a2)


# --------------------------------------------------------------------------------------------------
# 4) Resolve exact historical frozen decision object
# --------------------------------------------------------------------------------------------------

frozen_decision_name_628hc7a2 = (
    "FD002_FROZEN_ADAPTIVE_DECISION_628HC3A3F2B"
)

if frozen_decision_name_628hc7a2 not in G_628hc7a2:
    raise RuntimeError(
        "Expected historical frozen adaptive-decision table is missing."
    )

frozen_decision_628hc7a2 = G_628hc7a2[
    frozen_decision_name_628hc7a2
]

print("\n--- FROZEN ADAPTIVE DECISION ---")
display(frozen_decision_628hc7a2)


# --------------------------------------------------------------------------------------------------
# 5) Resolve exact frozen selected CLEAN risk
# --------------------------------------------------------------------------------------------------

clean_frozen_name_628hc7a2 = (
    "FD002_FROZEN_SELECTED_CLEAN_RISK_628HC3A3F2B"
)

if clean_frozen_name_628hc7a2 not in G_628hc7a2:
    raise RuntimeError(
        "Historical frozen selected CLEAN risk is missing."
    )

frozen_clean_risk_628hc7a2 = np.asarray(
    G_628hc7a2[
        clean_frozen_name_628hc7a2
    ],
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 6) Resolve exact frozen selected CORRUPTED risk without guessing suffix
# --------------------------------------------------------------------------------------------------

corrupted_frozen_candidates_628hc7a2 = sorted(
    [
        name
        for name in frozen_names_628hc7a2
        if name.startswith(
            "FD002_FROZEN_SELECTED_CORRUPTED_RISK_"
        )
    ]
)

print("\n--- FROZEN CORRUPTED-RISK CANDIDATES ---")
print(corrupted_frozen_candidates_628hc7a2)

if len(corrupted_frozen_candidates_628hc7a2) != 1:
    raise RuntimeError(
        "Expected exactly one frozen selected corrupted-risk object; "
        f"found {len(corrupted_frozen_candidates_628hc7a2)}."
    )

corrupted_frozen_name_628hc7a2 = (
    corrupted_frozen_candidates_628hc7a2[0]
)

frozen_corrupted_risk_628hc7a2 = np.asarray(
    G_628hc7a2[
        corrupted_frozen_name_628hc7a2
    ],
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 7) Frozen matrix contract
# --------------------------------------------------------------------------------------------------

clean_shape_ok_628hc7a2 = bool(
    frozen_clean_risk_628hc7a2.shape
    == (EXPECTED_WINDOWS_628hc7a2,)
)

corrupted_shape_ok_628hc7a2 = bool(
    frozen_corrupted_risk_628hc7a2.shape
    == (
        EXPECTED_SCENARIOS_628hc7a2,
        EXPECTED_WINDOWS_628hc7a2,
    )
)

clean_finite_ok_628hc7a2 = bool(
    np.isfinite(
        frozen_clean_risk_628hc7a2
    ).all()
)

corrupted_finite_ok_628hc7a2 = bool(
    np.isfinite(
        frozen_corrupted_risk_628hc7a2
    ).all()
)

clean_range_ok_628hc7a2 = bool(
    clean_finite_ok_628hc7a2
    and np.min(frozen_clean_risk_628hc7a2) >= 0.0
    and np.max(frozen_clean_risk_628hc7a2) <= 1.0
)

corrupted_range_ok_628hc7a2 = bool(
    corrupted_finite_ok_628hc7a2
    and np.min(frozen_corrupted_risk_628hc7a2) >= 0.0
    and np.max(frozen_corrupted_risk_628hc7a2) <= 1.0
)


print("\n--- FROZEN SELECTED RISK CONTRACT ---")

print(
    "Clean object             :",
    clean_frozen_name_628hc7a2,
)
print(
    "Clean shape              :",
    frozen_clean_risk_628hc7a2.shape,
)
print(
    "Clean min / median / max :",
    float(np.min(frozen_clean_risk_628hc7a2)),
    float(np.median(frozen_clean_risk_628hc7a2)),
    float(np.max(frozen_clean_risk_628hc7a2)),
)

print()

print(
    "Corrupted object         :",
    corrupted_frozen_name_628hc7a2,
)
print(
    "Corrupted shape          :",
    frozen_corrupted_risk_628hc7a2.shape,
)
print(
    "Corrupted min/median/max :",
    float(np.min(frozen_corrupted_risk_628hc7a2)),
    float(np.median(frozen_corrupted_risk_628hc7a2)),
    float(np.max(frozen_corrupted_risk_628hc7a2)),
)


# --------------------------------------------------------------------------------------------------
# 8) Compare historical frozen selected risk with uncertainty-only baseline
#
# This does NOT select anything.
# It only identifies whether the already-frozen architecture differs from the
# uncertainty-only baseline.
# --------------------------------------------------------------------------------------------------

clean_uncertainty_628hc7a2 = np.asarray(
    G_628hc7a2[
        "clean_uncertainty_risk_628hc3a0r1"
    ],
    dtype=np.float64,
)

corrupted_uncertainty_628hc7a2 = np.asarray(
    G_628hc7a2[
        "uncertainty_risk_matrix_628hc3a0r1"
    ],
    dtype=np.float64,
)

clean_vs_uncertainty_maxerr_628hc7a2 = float(
    np.max(
        np.abs(
            frozen_clean_risk_628hc7a2
            - clean_uncertainty_628hc7a2
        )
    )
)

corrupted_vs_uncertainty_maxerr_628hc7a2 = float(
    np.max(
        np.abs(
            frozen_corrupted_risk_628hc7a2
            - corrupted_uncertainty_628hc7a2
        )
    )
)

clean_equals_uncertainty_628hc7a2 = bool(
    np.array_equal(
        frozen_clean_risk_628hc7a2,
        clean_uncertainty_628hc7a2,
    )
)

corrupted_equals_uncertainty_628hc7a2 = bool(
    np.array_equal(
        frozen_corrupted_risk_628hc7a2,
        corrupted_uncertainty_628hc7a2,
    )
)

print("\n--- FROZEN SELECTION VS UNCERTAINTY-ONLY BASELINE ---")

print(
    "Clean exact equality                   :",
    clean_equals_uncertainty_628hc7a2,
)

print(
    "Clean max abs difference               :",
    clean_vs_uncertainty_maxerr_628hc7a2,
)

print(
    "Corrupted exact equality               :",
    corrupted_equals_uncertainty_628hc7a2,
)

print(
    "Corrupted global max abs difference    :",
    corrupted_vs_uncertainty_maxerr_628hc7a2,
)


# --------------------------------------------------------------------------------------------------
# 9) Audit
# --------------------------------------------------------------------------------------------------

audit_628hc7a2 = pd.DataFrame(
    {
        "check": [
            "previous_C7_A1_passed",
            "historical_A7_decision_objects_present",
            "frozen_decision_table_present",
            "frozen_clean_risk_shape_8367",
            "frozen_corrupted_risk_shape_315x8367",
            "frozen_clean_risk_finite",
            "frozen_corrupted_risk_finite",
            "frozen_clean_risk_in_0_1",
            "frozen_corrupted_risk_in_0_1",
            "no_new_selection_performed",
            "no_model_execution",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_test_access_or_scoring",
            "no_threshold_or_rule_modification",
            "no_files_written",
        ],
        "passed": [
            previous_ok_628hc7a2,
            decision_objects_present_628hc7a2,
            isinstance(
                frozen_decision_628hc7a2,
                pd.DataFrame,
            ),
            clean_shape_ok_628hc7a2,
            corrupted_shape_ok_628hc7a2,
            clean_finite_ok_628hc7a2,
            corrupted_finite_ok_628hc7a2,
            clean_range_ok_628hc7a2,
            corrupted_range_ok_628hc7a2,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)

print("\n--- C7-A2 AUDIT ---")
display(audit_628hc7a2)

all_checks_passed_628hc7a2 = bool(
    audit_628hc7a2[
        "passed"
    ].all()
)

if not all_checks_passed_628hc7a2:
    raise RuntimeError(
        "C7-A2 historical freeze recovery failed. "
        "Do not proceed to FD002 TEST preparation."
    )


# --------------------------------------------------------------------------------------------------
# 10) Authoritative recovered-state aliases
#
# These are aliases only. No rule is recomputed.
# --------------------------------------------------------------------------------------------------

FD002_RECOVERED_FROZEN_CLEAN_RISK_628HC7A2 = (
    frozen_clean_risk_628hc7a2
)

FD002_RECOVERED_FROZEN_CORRUPTED_RISK_628HC7A2 = (
    frozen_corrupted_risk_628hc7a2
)

FD002_RECOVERED_SELECTED_ARCHITECTURE_628HC7A2 = (
    historical_selected_architecture_628hc7a2
)

FD002_RECOVERED_ADAPTIVE_ACCEPTED_628HC7A2 = (
    historical_adaptive_accepted_628hc7a2
)


# --------------------------------------------------------------------------------------------------
# 11) Final summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 122)
print("STEP 6.28H-C7-A2 FINAL SUMMARY")
print("=" * 122)

print(
    "Historical adaptive fusion accepted      :",
    historical_adaptive_accepted_628hc7a2,
)

print(
    "Historical selected architecture         :",
    historical_selected_architecture_628hc7a2,
)

print(
    "Historical decision text                 :",
    historical_decision_text_628hc7a2,
)

print(
    "Frozen clean risk shape                  :",
    frozen_clean_risk_628hc7a2.shape,
)

print(
    "Frozen corrupted risk shape              :",
    frozen_corrupted_risk_628hc7a2.shape,
)

print(
    "Frozen clean == uncertainty-only         :",
    clean_equals_uncertainty_628hc7a2,
)

print(
    "Frozen corrupted == uncertainty-only     :",
    corrupted_equals_uncertainty_628hc7a2,
)

print(
    "Model executed                           : False"
)

print(
    "MC dropout executed                      : False"
)

print(
    "Model retrained                          : False"
)

print(
    "FD002 TEST accessed/scored               : False"
)

print(
    "Threshold/rule modified                  : False"
)

print(
    "Files written                            : False"
)

print(
    "All C7-A2 contract checks passed         :",
    all_checks_passed_628hc7a2,
)

print(
    "\nPASS — the already-completed historical FD002 adaptive-selection/freeze "
    "has been recovered without performing a new selection."
)

print(
    "Next: lock this recovered FD002 reliability construction as the authoritative "
    "pre-TEST policy, then build the FD002 TEST execution contract."
)

print("\n=== END STEP 6.28H-C7-A2 ===")

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A2-R1 — FD002 AUTHORITATIVE HISTORICAL FREEZE RECOVERY
#
# PURPOSE
# -------
# Recover the exact historical FD002 decision and the later authoritative C4 frozen policy.
#
# READ-ONLY:
#   - no new selection
#   - no threshold tuning
#   - no model execution
#   - no MC dropout
#   - no retraining
#   - no TEST access/scoring
#   - no file writing
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C7-A2-R1 — FD002 AUTHORITATIVE HISTORICAL FREEZE RECOVERY")
print("=" * 122)

G = dict(globals())


# --------------------------------------------------------------------------------------------------
# 1) Previous recovery contract
# --------------------------------------------------------------------------------------------------

if not bool(
    G.get(
        "all_checks_passed_628hc7a1",
        False,
    )
):
    raise RuntimeError(
        "C7-A1 must pass before C7-A2-R1."
    )


# --------------------------------------------------------------------------------------------------
# 2) Exact-prefix resolver
#
# Avoids hard-coding historical suffixes.
# --------------------------------------------------------------------------------------------------

def resolve_one(prefix):

    matches = sorted(
        [
            name
            for name in G
            if name.startswith(prefix)
        ]
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"Expected exactly one object with prefix {prefix!r}; "
            f"found {len(matches)}: {matches}"
        )

    return matches[0], G[matches[0]]


# --------------------------------------------------------------------------------------------------
# 3) Historical C3-A3 adaptive decision
# --------------------------------------------------------------------------------------------------

hist_decision_name, hist_decision = resolve_one(
    "FD002_FROZEN_ADAPTIVE_DECISION_"
)

adaptive_accepted = bool(
    G[
        "FD002_C3A3A3A7_ADAPTIVE_FUSION_ACCEPTED"
    ]
)

decision_text = str(
    G[
        "FD002_C3A3A3A7_DECISION"
    ]
)

selected_architecture = str(
    G[
        "FD002_C3A3A3A7_SELECTED_ARCHITECTURE"
    ]
)


print("\n--- HISTORICAL C3-A3 DECISION ---")

print(
    "Frozen decision object  :",
    hist_decision_name,
)

print(
    "Adaptive fusion accepted:",
    adaptive_accepted,
)

print(
    "Decision text            :",
    decision_text,
)

print(
    "Selected architecture    :",
    selected_architecture,
)

display(
    hist_decision
)


# --------------------------------------------------------------------------------------------------
# 4) Recover later C4 frozen policy
#
# This is chronologically later and therefore the authoritative FD002 policy freeze.
# --------------------------------------------------------------------------------------------------

c4_policy_name_obj, c4_policy_name = resolve_one(
    "FD002_FROZEN_C4_SELECTED_POLICY_NAME_"
)

c4_threshold_name, c4_threshold_obj = resolve_one(
    "FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_"
)

c4_target_name, c4_target_obj = resolve_one(
    "FD002_FROZEN_C4_SELECTED_TARGET_COVERAGE_"
)

c4_metadata_name, c4_metadata = resolve_one(
    "FD002_FROZEN_C4_POLICY_METADATA_"
)

c4_audit_name, c4_audit = resolve_one(
    "FD002_FROZEN_C4_POLICY_SELECTION_AUDIT_"
)


c4_policy_name = str(
    c4_policy_name
)

c4_threshold = float(
    c4_threshold_obj
)

c4_target_coverage = float(
    c4_target_obj
)


print("\n--- AUTHORITATIVE C4 FROZEN POLICY ---")

print(
    "Policy-name object       :",
    c4_policy_name_obj,
)

print(
    "Selected policy          :",
    c4_policy_name,
)

print(
    "Frozen risk threshold    :",
    c4_threshold,
)

print(
    "Frozen target coverage   :",
    c4_target_coverage,
)

print(
    "Metadata object          :",
    c4_metadata_name,
)

print(
    "Selection-audit object   :",
    c4_audit_name,
)


print("\n--- C4 POLICY METADATA ---")
print(
    c4_metadata
)


print("\n--- C4 POLICY SELECTION AUDIT ---")
display(
    c4_audit
)


# --------------------------------------------------------------------------------------------------
# 5) Recover frozen selected risk arrays
# --------------------------------------------------------------------------------------------------

clean_name, frozen_clean = resolve_one(
    "FD002_FROZEN_SELECTED_CLEAN_RISK_"
)

corrupted_name, frozen_corrupted = resolve_one(
    "FD002_FROZEN_SELECTED_CORRUPTED_RISK_"
)


frozen_clean = np.asarray(
    frozen_clean,
    dtype=np.float64,
)

frozen_corrupted = np.asarray(
    frozen_corrupted,
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 6) Recover uncertainty-only reference
# --------------------------------------------------------------------------------------------------

unc_clean = np.asarray(
    G[
        "clean_uncertainty_risk_628hc3a0r1"
    ],
    dtype=np.float64,
)

unc_corrupted = np.asarray(
    G[
        "uncertainty_risk_matrix_628hc3a0r1"
    ],
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 7) Frozen selected-risk contract
# --------------------------------------------------------------------------------------------------

clean_shape_ok = bool(
    frozen_clean.shape
    ==
    (8367,)
)

corrupted_shape_ok = bool(
    frozen_corrupted.shape
    ==
    (315, 8367)
)

clean_finite = bool(
    np.isfinite(
        frozen_clean
    ).all()
)

corrupted_finite = bool(
    np.isfinite(
        frozen_corrupted
    ).all()
)

clean_range = bool(
    clean_finite
    and
    frozen_clean.min() >= 0.0
    and
    frozen_clean.max() <= 1.0
)

corrupted_range = bool(
    corrupted_finite
    and
    frozen_corrupted.min() >= 0.0
    and
    frozen_corrupted.max() <= 1.0
)


# --------------------------------------------------------------------------------------------------
# 8) Verify frozen selected risk is uncertainty-only
# --------------------------------------------------------------------------------------------------

clean_equals_unc = bool(
    np.array_equal(
        frozen_clean,
        unc_clean,
    )
)

corrupted_equals_unc = bool(
    np.array_equal(
        frozen_corrupted,
        unc_corrupted,
    )
)


clean_max_abs_diff = float(
    np.max(
        np.abs(
            frozen_clean
            -
            unc_clean
        )
    )
)

corrupted_max_abs_diff = float(
    np.max(
        np.abs(
            frozen_corrupted
            -
            unc_corrupted
        )
    )
)


print("\n--- FROZEN SELECTED-RISK IDENTITY ---")

print(
    "Clean frozen object        :",
    clean_name,
)

print(
    "Corrupted frozen object    :",
    corrupted_name,
)

print(
    "Clean shape                :",
    frozen_clean.shape,
)

print(
    "Corrupted shape            :",
    frozen_corrupted.shape,
)

print(
    "Clean == uncertainty-only  :",
    clean_equals_unc,
)

print(
    "Corrupt == uncertainty-only:",
    corrupted_equals_unc,
)

print(
    "Clean max abs difference   :",
    clean_max_abs_diff,
)

print(
    "Corrupt max abs difference :",
    corrupted_max_abs_diff,
)


# --------------------------------------------------------------------------------------------------
# 9) Historical consistency checks
# --------------------------------------------------------------------------------------------------

adaptive_rejected_consistent = bool(
    adaptive_accepted is False
    and
    decision_text
    ==
    "retain_uncertainty_only"
)

policy_name_consistent = bool(
    selected_architecture
    ==
    "uncertainty_only"
    and
    c4_policy_name
    ==
    "uncertainty_only_fixed_threshold"
)

risk_identity_consistent = bool(
    clean_equals_unc
    and
    corrupted_equals_unc
)


# --------------------------------------------------------------------------------------------------
# 10) Audit
# --------------------------------------------------------------------------------------------------

audit = pd.DataFrame(
    {
        "check": [
            "previous_C7_A1_passed",
            "historical_frozen_decision_resolved",
            "adaptive_fusion_rejected_consistently",
            "C4_frozen_policy_resolved",
            "C4_policy_matches_uncertainty_only",
            "frozen_clean_shape_8367",
            "frozen_corrupted_shape_315x8367",
            "frozen_clean_finite_in_0_1",
            "frozen_corrupted_finite_in_0_1",
            "frozen_selected_risks_equal_uncertainty_only",
            "no_new_selection",
            "no_model_execution",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_test_access_or_scoring",
            "no_threshold_or_rule_modification",
            "no_files_written",
        ],

        "passed": [
            True,
            isinstance(
                hist_decision,
                pd.DataFrame,
            ),
            adaptive_rejected_consistent,
            True,
            policy_name_consistent,
            clean_shape_ok,
            corrupted_shape_ok,
            clean_range,
            corrupted_range,
            risk_identity_consistent,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)


print("\n--- C7-A2-R1 AUDIT ---")
display(
    audit
)


all_checks_passed_628hc7a2r1 = bool(
    audit[
        "passed"
    ].all()
)


if not all_checks_passed_628hc7a2r1:

    raise RuntimeError(
        "C7-A2-R1 authoritative frozen-policy recovery did not pass. "
        "Do not proceed to FD002 TEST."
    )


# --------------------------------------------------------------------------------------------------
# 11) Authoritative read-only aliases
# --------------------------------------------------------------------------------------------------

FD002_AUTHORITATIVE_POLICY_NAME_628HC7A2R1 = (
    c4_policy_name
)

FD002_AUTHORITATIVE_THRESHOLD_628HC7A2R1 = (
    c4_threshold
)

FD002_AUTHORITATIVE_TARGET_COVERAGE_628HC7A2R1 = (
    c4_target_coverage
)

FD002_AUTHORITATIVE_CLEAN_RISK_628HC7A2R1 = (
    frozen_clean
)

FD002_AUTHORITATIVE_CORRUPTED_RISK_628HC7A2R1 = (
    frozen_corrupted
)


# --------------------------------------------------------------------------------------------------
# 12) Final summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 122)
print("STEP 6.28H-C7-A2-R1 FINAL SUMMARY")
print("=" * 122)

print(
    "Historical adaptive fusion accepted :",
    adaptive_accepted,
)

print(
    "Historical decision                 :",
    decision_text,
)

print(
    "Historical selected architecture    :",
    selected_architecture,
)

print(
    "Authoritative C4 policy             :",
    c4_policy_name,
)

print(
    "Frozen risk threshold               :",
    c4_threshold,
)

print(
    "Target coverage                     :",
    c4_target_coverage,
)

print(
    "Selected clean risk == uncertainty  :",
    clean_equals_unc,
)

print(
    "Selected corrupt risk == uncertainty:",
    corrupted_equals_unc,
)

print(
    "Model executed                      : False"
)

print(
    "MC dropout executed                 : False"
)

print(
    "Model retrained                     : False"
)

print(
    "FD002 TEST accessed/scored          : False"
)

print(
    "Threshold/rule modified             : False"
)

print(
    "Files written                       : False"
)

print(
    "All C7-A2-R1 checks passed          :",
    all_checks_passed_628hc7a2r1,
)


print(
    "\nPASS — the FD002 historical selection is recovered. "
    "Adaptive fusion was rejected; the later C4 freeze confirms that the "
    "authoritative policy is uncertainty-only with the already-frozen fixed threshold."
)

print(
    "Next: lock the exact FD002 TEST input/execution contract using this frozen C4 policy. "
    "No DEV retuning is permitted."
)

print(
    "\n=== END STEP 6.28H-C7-A2-R1 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A3 — FD002 FROZEN ARTIFACT RELOAD / HASH VERIFICATION (PRE-TEST)
#
# PURPOSE
# -------
# Verify that the already-frozen FD002 reliability/policy artifacts on disk can be found, hashed,
# reloaded, and matched to the authoritative in-memory freeze BEFORE any FD002 TEST access.
#
# READ-ONLY:
#   - no DEV retuning
#   - no TEST access/scoring
#   - no model execution
#   - no MC dropout
#   - no retraining
#   - no threshold/rule modification
#   - no files written
# ==================================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C7-A3 — FD002 FROZEN ARTIFACT RELOAD / HASH VERIFICATION (PRE-TEST)")
print("=" * 122)


# --------------------------------------------------------------------------------------------------
# 1) Previous authoritative-recovery lock
# --------------------------------------------------------------------------------------------------

if not bool(
    globals().get(
        "all_checks_passed_628hc7a2r1",
        False,
    )
):
    raise RuntimeError(
        "C7-A2-R1 must pass before C7-A3."
    )


metadata = globals().get(
    "FD002_FROZEN_C4_POLICY_METADATA_628HC4F1B"
)

if not isinstance(
    metadata,
    dict,
):
    raise RuntimeError(
        "Authoritative C4 policy metadata object is missing."
    )


policy_name = str(
    globals()[
        "FD002_AUTHORITATIVE_POLICY_NAME_628HC7A2R1"
    ]
)

threshold = float(
    globals()[
        "FD002_AUTHORITATIVE_THRESHOLD_628HC7A2R1"
    ]
)

target_coverage = float(
    globals()[
        "FD002_AUTHORITATIVE_TARGET_COVERAGE_628HC7A2R1"
    ]
)

clean_risk_memory = np.asarray(
    globals()[
        "FD002_AUTHORITATIVE_CLEAN_RISK_628HC7A2R1"
    ],
    dtype=np.float64,
)

corrupted_risk_memory = np.asarray(
    globals()[
        "FD002_AUTHORITATIVE_CORRUPTED_RISK_628HC7A2R1"
    ],
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 2) Resolve repository root without touching TEST data
# --------------------------------------------------------------------------------------------------

cwd = Path.cwd().resolve()

root_candidates = [
    cwd,
] + list(
    cwd.parents
)

repo_root = None

for p in root_candidates:

    if (
        p
        /
        "02_dataset"
    ).exists():

        repo_root = p
        break


if repo_root is None:

    raise RuntimeError(
        "Could not resolve repository root from the current notebook working directory."
    )


print(
    "\n--- REPOSITORY ROOT ---"
)

print(
    "Notebook cwd :",
    cwd,
)

print(
    "Repository   :",
    repo_root,
)


# --------------------------------------------------------------------------------------------------
# 3) SHA-256 helper
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            block = f.read(
                1024
                *
                1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4) Verify upstream frozen reliability NPZ by exact filename + SHA-256
# --------------------------------------------------------------------------------------------------

upstream = metadata.get(
    "upstream_reliability_artifact",
    {},
)

npz_filename = str(
    upstream.get(
        "filename",
        "",
    )
)

npz_expected_sha = str(
    upstream.get(
        "sha256",
        "",
    )
)


if (
    not npz_filename
    or
    not npz_expected_sha
):

    raise RuntimeError(
        "C4 metadata does not contain the upstream reliability artifact filename/SHA-256."
    )


npz_matches = sorted(
    repo_root.rglob(
        npz_filename
    )
)


if len(
    npz_matches
) != 1:

    raise RuntimeError(
        f"Expected exactly one {npz_filename!r} under {repo_root}; "
        f"found {len(npz_matches)}: {npz_matches}"
    )


npz_path = npz_matches[
    0
]

npz_actual_sha = sha256_file(
    npz_path
)

npz_hash_ok = bool(
    npz_actual_sha
    ==
    npz_expected_sha
)


if not npz_hash_ok:

    raise RuntimeError(
        "Upstream FD002 reliability NPZ SHA-256 does not match the frozen metadata."
    )


print(
    "\n--- UPSTREAM RELIABILITY ARTIFACT ---"
)

print(
    "Path          :",
    npz_path,
)

print(
    "Expected SHA  :",
    npz_expected_sha,
)

print(
    "Actual SHA    :",
    npz_actual_sha,
)

print(
    "SHA exact     :",
    npz_hash_ok,
)


# --------------------------------------------------------------------------------------------------
# 5) Reload NPZ and resolve clean/corrupted risk arrays by exact frozen shapes
# --------------------------------------------------------------------------------------------------

npz = np.load(
    npz_path,
    allow_pickle=False,
)

npz_keys = list(
    npz.files
)


clean_candidates = []
corrupted_candidates = []


for key in npz_keys:

    arr = np.asarray(
        npz[
            key
        ]
    )

    if arr.shape == (
        8367,
    ):

        clean_candidates.append(
            (
                key,
                arr,
            )
        )

    if arr.shape == (
        315,
        8367,
    ):

        corrupted_candidates.append(
            (
                key,
                arr,
            )
        )


if len(
    clean_candidates
) != 1:

    npz.close()

    raise RuntimeError(
        "Expected exactly one clean-risk array with shape (8367,) in the frozen NPZ; "
        f"found {len(clean_candidates)} keys: {[x[0] for x in clean_candidates]}"
    )


if len(
    corrupted_candidates
) != 1:

    npz.close()

    raise RuntimeError(
        "Expected exactly one corrupted-risk array with shape (315, 8367) in the frozen NPZ; "
        f"found {len(corrupted_candidates)} keys: {[x[0] for x in corrupted_candidates]}"
    )


clean_key, clean_risk_disk_raw = (
    clean_candidates[
        0
    ]
)

corrupted_key, corrupted_risk_disk_raw = (
    corrupted_candidates[
        0
    ]
)


clean_risk_disk = np.asarray(
    clean_risk_disk_raw,
    dtype=np.float64,
).copy()

corrupted_risk_disk = np.asarray(
    corrupted_risk_disk_raw,
    dtype=np.float64,
).copy()


npz.close()


clean_reload_exact = bool(
    np.array_equal(
        clean_risk_disk,
        clean_risk_memory,
    )
)

corrupted_reload_exact = bool(
    np.array_equal(
        corrupted_risk_disk,
        corrupted_risk_memory,
    )
)


clean_reload_max_abs = float(
    np.max(
        np.abs(
            clean_risk_disk
            -
            clean_risk_memory
        )
    )
)

corrupted_reload_max_abs = float(
    np.max(
        np.abs(
            corrupted_risk_disk
            -
            corrupted_risk_memory
        )
    )
)


print(
    "\n--- NPZ RELOAD IDENTITY ---"
)

print(
    "NPZ keys                  :",
    npz_keys,
)

print(
    "Resolved clean-risk key   :",
    clean_key,
)

print(
    "Resolved corrupted key    :",
    corrupted_key,
)

print(
    "Clean reload exact        :",
    clean_reload_exact,
)

print(
    "Corrupt reload exact      :",
    corrupted_reload_exact,
)

print(
    "Clean max abs difference  :",
    clean_reload_max_abs,
)

print(
    "Corrupt max abs difference:",
    corrupted_reload_max_abs,
)


# --------------------------------------------------------------------------------------------------
# 6) Verify the two C4 frozen CSV artifacts using metadata SHA-256 values
# --------------------------------------------------------------------------------------------------

frozen_file_sha = metadata.get(
    "frozen_file_sha256",
    {},
)


expected_csv_hashes = {
    "policy_selection_audit_csv": str(
        frozen_file_sha.get(
            "policy_selection_audit_csv",
            "",
        )
    ),

    "selected_family_severity_csv": str(
        frozen_file_sha.get(
            "selected_family_severity_csv",
            "",
        )
    ),
}


if not all(
    expected_csv_hashes.values()
):

    raise RuntimeError(
        "C4 metadata is missing one or more frozen CSV SHA-256 values."
    )


csv_paths = sorted(
    repo_root.rglob(
        "*.csv"
    )
)


hash_to_paths = {}


for p in csv_paths:

    digest = sha256_file(
        p
    )

    hash_to_paths.setdefault(
        digest,
        [],
    ).append(
        p
    )


policy_csv_matches = hash_to_paths.get(
    expected_csv_hashes[
        "policy_selection_audit_csv"
    ],
    [],
)

family_csv_matches = hash_to_paths.get(
    expected_csv_hashes[
        "selected_family_severity_csv"
    ],
    [],
)


policy_csv_unique = bool(
    len(
        policy_csv_matches
    )
    ==
    1
)

family_csv_unique = bool(
    len(
        family_csv_matches
    )
    ==
    1
)


print(
    "\n--- C4 FROZEN CSV HASH VERIFICATION ---"
)

print(
    "Policy-selection audit matches :",
    policy_csv_matches,
)

print(
    "Family×severity matches        :",
    family_csv_matches,
)


# --------------------------------------------------------------------------------------------------
# 7) Frozen policy contract
# --------------------------------------------------------------------------------------------------

policy_name_ok = bool(
    policy_name
    ==
    "uncertainty_only_fixed_threshold"
)


threshold_ok = bool(
    np.isfinite(
        threshold
    )
    and
    0.0
    <=
    threshold
    <=
    1.0
    and
    threshold
    ==
    float(
        metadata[
            "selected_exact_risk_threshold"
        ]
    )
)


coverage_ok = bool(
    target_coverage
    ==
    0.8
    and
    target_coverage
    ==
    float(
        metadata[
            "selected_nominal_coverage"
        ]
    )
)


method_status = metadata.get(
    "methodological_status",
    {},
)


metadata_pretest_ok = bool(
    method_status.get(
        "official_test_labels_read",
        None,
    )
    is False
    and
    method_status.get(
        "test_scored",
        None,
    )
    is False
    and
    method_status.get(
        "threshold_selected_before_test",
        None,
    )
    is True
)


# --------------------------------------------------------------------------------------------------
# 8) Audit
# --------------------------------------------------------------------------------------------------

audit = pd.DataFrame(
    {
        "check": [
            "previous_C7_A2_R1_passed",
            "authoritative_policy_uncertainty_only_fixed_threshold",
            "threshold_matches_frozen_metadata",
            "target_coverage_matches_frozen_metadata",
            "upstream_npz_unique_path",
            "upstream_npz_sha256_exact",
            "clean_risk_reload_exact",
            "corrupted_risk_reload_exact",
            "policy_selection_csv_sha256_unique_match",
            "family_severity_csv_sha256_unique_match",
            "metadata_confirms_threshold_selected_before_test",
            "no_dev_retuning",
            "no_test_access_or_scoring",
            "no_model_execution",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_threshold_or_rule_modification",
            "no_files_written",
        ],

        "passed": [
            True,
            policy_name_ok,
            threshold_ok,
            coverage_ok,
            len(
                npz_matches
            )
            ==
            1,
            npz_hash_ok,
            clean_reload_exact,
            corrupted_reload_exact,
            policy_csv_unique,
            family_csv_unique,
            metadata_pretest_ok,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
        ],
    }
)


print(
    "\n--- C7-A3 AUDIT ---"
)

display(
    audit
)


all_checks_passed_628hc7a3 = bool(
    audit[
        "passed"
    ].all()
)


if not all_checks_passed_628hc7a3:

    raise RuntimeError(
        "C7-A3 frozen-artifact reload/hash verification did not pass. "
        "Do not access FD002 TEST."
    )


# --------------------------------------------------------------------------------------------------
# 9) Read-only verification record
# --------------------------------------------------------------------------------------------------

FD002_C7A3_VERIFIED_POLICY_628HC7A3 = {
    "policy_name":
        policy_name,

    "risk_threshold":
        threshold,

    "target_coverage":
        target_coverage,

    "upstream_npz_path":
        str(
            npz_path
        ),

    "upstream_npz_sha256":
        npz_actual_sha,

    "clean_risk_key":
        clean_key,

    "corrupted_risk_key":
        corrupted_key,

    "policy_selection_csv_path":
        str(
            policy_csv_matches[
                0
            ]
        ),

    "selected_family_severity_csv_path":
        str(
            family_csv_matches[
                0
            ]
        ),

    "verification_passed":
        True,
}


# --------------------------------------------------------------------------------------------------
# 10) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 122
)

print(
    "STEP 6.28H-C7-A3 FINAL SUMMARY"
)

print(
    "=" * 122
)

print(
    "Policy name                     :",
    policy_name,
)

print(
    "Frozen risk threshold           :",
    threshold,
)

print(
    "Target coverage                 :",
    target_coverage,
)

print(
    "Upstream NPZ SHA verified       :",
    npz_hash_ok,
)

print(
    "Clean risk reload exact         :",
    clean_reload_exact,
)

print(
    "Corrupted risk reload exact     :",
    corrupted_reload_exact,
)

print(
    "Policy CSV hash verified        :",
    policy_csv_unique,
)

print(
    "Family×severity CSV verified    :",
    family_csv_unique,
)

print(
    "FD002 TEST accessed/scored      : False"
)

print(
    "Model/MC executed               : False"
)

print(
    "DEV retuning                    : False"
)

print(
    "Threshold/rule modified         : False"
)

print(
    "Files written                   : False"
)

print(
    "All C7-A3 checks passed         :",
    all_checks_passed_628hc7a3,
)


print(
    "\nPASS — the frozen FD002 policy is now verified against its persisted artifacts. "
    "No TEST data were accessed."
)

print(
    "Next: lock the exact FD002 TEST feature/RUL source paths and TEST execution contract, "
    "then run the frozen policy one-way with no retuning."
)

print(
    "\n=== END STEP 6.28H-C7-A3 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A4 — FD002 EXACT TEST / RUL SOURCE-PATH + EXECUTION CONTRACT LOCK
#
# PURPOSE
# -------
# Lock the unique official FD002 TEST-feature and RUL source paths and bind them to the already-
# frozen FD002 policy before reading either file.
#
# SAFETY
# ------
# - TEST feature contents NOT read
# - official RUL contents NOT read
# - no TEST scoring
# - no DEV retuning
# - no model execution
# - no MC dropout execution
# - no retraining
# - no threshold/rule modification
# - no files written
# ==================================================================================================

from pathlib import Path
import pandas as pd

print("=" * 122)
print("STEP 6.28H-C7-A4 — FD002 EXACT TEST / RUL SOURCE-PATH + EXECUTION CONTRACT LOCK")
print("=" * 122)


# --------------------------------------------------------------------------------------------------
# 1) Previous frozen-artifact verification must already have passed
# --------------------------------------------------------------------------------------------------

if not bool(
    globals().get(
        "all_checks_passed_628hc7a3",
        False,
    )
):
    raise RuntimeError(
        "C7-A3 must pass before C7-A4."
    )


verified_policy_628hc7a4 = globals().get(
    "FD002_C7A3_VERIFIED_POLICY_628HC7A3"
)

if not isinstance(
    verified_policy_628hc7a4,
    dict,
):
    raise RuntimeError(
        "C7-A3 verified-policy record is missing."
    )


policy_name_628hc7a4 = str(
    verified_policy_628hc7a4[
        "policy_name"
    ]
)

threshold_628hc7a4 = float(
    verified_policy_628hc7a4[
        "risk_threshold"
    ]
)

target_coverage_628hc7a4 = float(
    verified_policy_628hc7a4[
        "target_coverage"
    ]
)


# --------------------------------------------------------------------------------------------------
# 2) Resolve repository root
# --------------------------------------------------------------------------------------------------

cwd_628hc7a4 = Path.cwd().resolve()

root_candidates_628hc7a4 = (
    [cwd_628hc7a4]
    +
    list(
        cwd_628hc7a4.parents
    )
)

repo_root_628hc7a4 = None

for p_628hc7a4 in root_candidates_628hc7a4:

    if (
        p_628hc7a4
        /
        "02_dataset"
    ).exists():

        repo_root_628hc7a4 = p_628hc7a4
        break


if repo_root_628hc7a4 is None:

    raise RuntimeError(
        "Could not resolve repository root containing 02_dataset."
    )


# --------------------------------------------------------------------------------------------------
# 3) Resolve exact FD002 TEST/RUL paths by filename only
# --------------------------------------------------------------------------------------------------

test_matches_628hc7a4 = sorted(
    repo_root_628hc7a4.rglob(
        "test_FD002.txt"
    )
)

rul_matches_628hc7a4 = sorted(
    repo_root_628hc7a4.rglob(
        "RUL_FD002.txt"
    )
)


print(
    "\n--- EXACT FILESYSTEM MATCHES ---"
)

print(
    "Repository root :",
    repo_root_628hc7a4,
)

print(
    "TEST matches    :",
    len(
        test_matches_628hc7a4
    ),
)

for p_628hc7a4 in test_matches_628hc7a4:

    print(
        "  ",
        p_628hc7a4,
    )


print(
    "RUL matches     :",
    len(
        rul_matches_628hc7a4
    ),
)

for p_628hc7a4 in rul_matches_628hc7a4:

    print(
        "  ",
        p_628hc7a4,
    )


if len(
    test_matches_628hc7a4
) != 1:

    raise RuntimeError(
        f"Expected exactly one test_FD002.txt under {repo_root_628hc7a4}; "
        f"found {len(test_matches_628hc7a4)}."
    )


if len(
    rul_matches_628hc7a4
) != 1:

    raise RuntimeError(
        f"Expected exactly one RUL_FD002.txt under {repo_root_628hc7a4}; "
        f"found {len(rul_matches_628hc7a4)}."
    )


test_path_628hc7a4 = (
    test_matches_628hc7a4[
        0
    ]
)

rul_path_628hc7a4 = (
    rul_matches_628hc7a4[
        0
    ]
)


same_parent_628hc7a4 = bool(
    test_path_628hc7a4.parent
    ==
    rul_path_628hc7a4.parent
)

test_metadata_ok_628hc7a4 = bool(
    test_path_628hc7a4.is_file()
)

rul_metadata_ok_628hc7a4 = bool(
    rul_path_628hc7a4.is_file()
)


# --------------------------------------------------------------------------------------------------
# 4) Frozen execution contract
# --------------------------------------------------------------------------------------------------

print(
    "\n--- FROZEN POLICY EXECUTION CONTRACT ---"
)

print(
    "Policy name       :",
    policy_name_628hc7a4,
)

print(
    "Risk threshold    :",
    threshold_628hc7a4,
)

print(
    "Target coverage   :",
    target_coverage_628hc7a4,
)

print(
    "Comparator        : uncertainty_risk <= risk_threshold"
)

print(
    "TEST feature path :",
    test_path_628hc7a4,
)

print(
    "RUL path          :",
    rul_path_628hc7a4,
)

print(
    "Same source dir   :",
    same_parent_628hc7a4,
)


FD002_TEST_FEATURE_PATH_628HC7A4 = str(
    test_path_628hc7a4
)

FD002_RUL_PATH_628HC7A4 = str(
    rul_path_628hc7a4
)


FD002_TEST_EXECUTION_CONTRACT_628HC7A4 = {

    "dataset":
        "NASA C-MAPSS FD002",

    "policy_name":
        policy_name_628hc7a4,

    "reliability_architecture":
        "uncertainty_only",

    "risk_threshold":
        threshold_628hc7a4,

    "target_coverage":
        target_coverage_628hc7a4,

    "acceptance_rule":
        "accept iff uncertainty_risk <= risk_threshold",

    "test_feature_path":
        FD002_TEST_FEATURE_PATH_628HC7A4,

    "official_rul_path":
        FD002_RUL_PATH_628HC7A4,

    "test_feature_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "test_scored":
        False,

    "model_executed":
        False,

    "mc_dropout_executed":
        False,

    "model_retrained":
        False,

    "threshold_or_rule_modified":
        False,

    "files_written":
        False,
}


# --------------------------------------------------------------------------------------------------
# 5) Audit
# --------------------------------------------------------------------------------------------------

audit_628hc7a4 = pd.DataFrame(
    {
        "check": [

            "previous_C7_A3_passed",

            "policy_is_uncertainty_only_fixed_threshold",

            "threshold_matches_C7_A3",

            "target_coverage_0p80",

            "unique_test_FD002_path",

            "unique_RUL_FD002_path",

            "test_path_is_file_metadata_only",

            "rul_path_is_file_metadata_only",

            "test_and_rul_same_source_directory",

            "test_feature_contents_not_read",

            "official_rul_contents_not_read",

            "test_not_scored",

            "no_model_execution",

            "no_mc_dropout_execution",

            "no_model_retraining",

            "no_threshold_or_rule_modification",

            "no_files_written",
        ],

        "passed": [

            True,

            policy_name_628hc7a4
            ==
            "uncertainty_only_fixed_threshold",

            threshold_628hc7a4
            ==
            float(
                globals()[
                    "FD002_AUTHORITATIVE_THRESHOLD_628HC7A2R1"
                ]
            ),

            target_coverage_628hc7a4
            ==
            0.8,

            len(
                test_matches_628hc7a4
            )
            ==
            1,

            len(
                rul_matches_628hc7a4
            )
            ==
            1,

            test_metadata_ok_628hc7a4,

            rul_metadata_ok_628hc7a4,

            same_parent_628hc7a4,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)


print(
    "\n--- C7-A4 AUDIT ---"
)

display(
    audit_628hc7a4
)


all_checks_passed_628hc7a4 = bool(
    audit_628hc7a4[
        "passed"
    ].all()
)


if not all_checks_passed_628hc7a4:

    raise RuntimeError(
        "C7-A4 TEST/RUL source-path and execution-contract lock did not pass. "
        "Do not read FD002 TEST or RUL contents."
    )


# --------------------------------------------------------------------------------------------------
# 6) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 122
)

print(
    "STEP 6.28H-C7-A4 FINAL SUMMARY"
)

print(
    "=" * 122
)

print(
    "Frozen policy                  :",
    policy_name_628hc7a4,
)

print(
    "Frozen risk threshold          :",
    threshold_628hc7a4,
)

print(
    "Target coverage                :",
    target_coverage_628hc7a4,
)

print(
    "TEST feature source locked     :",
    True,
)

print(
    "Official RUL source locked     :",
    True,
)

print(
    "TEST feature contents read     :",
    False,
)

print(
    "Official RUL contents read     :",
    False,
)

print(
    "TEST scored                    :",
    False,
)

print(
    "Model/MC executed              :",
    False,
)

print(
    "DEV retuning                   :",
    False,
)

print(
    "Threshold/rule modified        :",
    False,
)

print(
    "Files written                  :",
    False,
)

print(
    "All C7-A4 checks passed        :",
    all_checks_passed_628hc7a4,
)


print(
    "\nPASS — exact FD002 TEST/RUL source paths and the frozen execution contract are locked "
    "without opening either file."
)

print(
    "Next: read FD002 TEST FEATURES once, recover the exact FD002 normalization/window bridge, "
    "and validate the TEST input tensor contract while keeping RUL_FD002.txt unopened."
)

print(
    "\n=== END STEP 6.28H-C7-A4 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A5 — FD002 OFFICIAL TEST FEATURE ONE-TIME READ
#                    + PREPROCESSING-CONTRACT RECOVERY
#
# PURPOSE
# -------
# 1) Read test_FD002.txt exactly once.
# 2) Validate the raw FD002 TEST feature contract.
# 3) Recover the exact historical FD002 normalization/window helpers from the current namespace.
#
# NO:
# - RUL_FD002.txt read
# - TEST scoring
# - model execution
# - MC dropout
# - retraining
# - DEV retuning
# - threshold/rule modification
# - files written
# ==================================================================================================

from pathlib import Path
import inspect
import numpy as np
import pandas as pd


print("=" * 126)
print(
    "STEP 6.28H-C7-A5 — FD002 OFFICIAL TEST FEATURE ONE-TIME READ "
    "+ PREPROCESSING-CONTRACT RECOVERY"
)
print("=" * 126)


# --------------------------------------------------------------------------------------------------
# 1) PRECONDITIONS
# --------------------------------------------------------------------------------------------------

if not bool(
    globals().get(
        "all_checks_passed_628hc7a4",
        False,
    )
):
    raise RuntimeError(
        "C7-A4 must pass before C7-A5."
    )


contract_628hc7a5 = globals().get(
    "FD002_TEST_EXECUTION_CONTRACT_628HC7A4"
)

if not isinstance(
    contract_628hc7a5,
    dict,
):
    raise RuntimeError(
        "FD002 C7-A4 execution contract is missing."
    )


test_path_628hc7a5 = Path(
    globals()[
        "FD002_TEST_FEATURE_PATH_628HC7A4"
    ]
).resolve()


rul_path_628hc7a5 = Path(
    globals()[
        "FD002_RUL_PATH_628HC7A4"
    ]
).resolve()


if not test_path_628hc7a5.is_file():

    raise RuntimeError(
        "Locked FD002 TEST feature path is not a file."
    )


if not rul_path_628hc7a5.is_file():

    raise RuntimeError(
        "Locked FD002 RUL path is not a file."
    )


# Prevent accidental repeated official TEST reads in this kernel.
if bool(
    globals().get(
        "FD002_OFFICIAL_TEST_FEATURES_READ_628HC7A5",
        False,
    )
):

    raise RuntimeError(
        "C7-A5 one-time TEST feature read has already occurred "
        "in this kernel. Do not read test_FD002.txt again."
    )


# --------------------------------------------------------------------------------------------------
# 2) RAW C-MAPSS FEATURE SCHEMA
# --------------------------------------------------------------------------------------------------

raw_columns_628hc7a5 = (

    [
        "unit_id",
        "cycle",
    ]

    +

    [
        f"op_setting_{i}"
        for i in range(
            1,
            4,
        )
    ]

    +

    [
        f"sensor_{i}"
        for i in range(
            1,
            22,
        )
    ]
)


if len(
    raw_columns_628hc7a5
) != 26:

    raise RuntimeError(
        "Internal FD002 raw-column contract is not 26 columns."
    )


# --------------------------------------------------------------------------------------------------
# 3) ONE-TIME OFFICIAL FD002 TEST FEATURE READ
# --------------------------------------------------------------------------------------------------

FD002_OFFICIAL_TEST_RAW_628HC7A5 = pd.read_csv(

    test_path_628hc7a5,

    sep=r"\s+",

    header=None,

    names=raw_columns_628hc7a5,

    engine="python",
)


FD002_OFFICIAL_TEST_FEATURES_READ_628HC7A5 = True


test_raw_628hc7a5 = (
    FD002_OFFICIAL_TEST_RAW_628HC7A5
)


# --------------------------------------------------------------------------------------------------
# 4) RAW TEST CONTRACT
# --------------------------------------------------------------------------------------------------

numeric_628hc7a5 = (
    test_raw_628hc7a5
    .to_numpy(
        dtype=np.float64
    )
)


raw_shape_ok_628hc7a5 = bool(

    test_raw_628hc7a5.shape

    ==

    (
        33991,
        26,
    )
)


raw_columns_ok_628hc7a5 = bool(

    list(
        test_raw_628hc7a5.columns
    )

    ==

    raw_columns_628hc7a5
)


finite_ok_628hc7a5 = bool(
    np.isfinite(
        numeric_628hc7a5
    ).all()
)


unit_numeric_628hc7a5 = (
    test_raw_628hc7a5[
        "unit_id"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


cycle_numeric_628hc7a5 = (
    test_raw_628hc7a5[
        "cycle"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


integer_ids_ok_628hc7a5 = bool(

    np.array_equal(

        unit_numeric_628hc7a5,

        np.floor(
            unit_numeric_628hc7a5
        ),
    )
)


integer_cycles_ok_628hc7a5 = bool(

    np.array_equal(

        cycle_numeric_628hc7a5,

        np.floor(
            cycle_numeric_628hc7a5
        ),
    )
)


engine_ids_628hc7a5 = (

    test_raw_628hc7a5[
        "unit_id"
    ]
    .astype(
        np.int64
    )
)


engine_count_628hc7a5 = int(
    engine_ids_628hc7a5.nunique()
)


engine_sequence_ok_628hc7a5 = bool(

    np.array_equal(

        np.sort(
            engine_ids_628hc7a5.unique()
        ),

        np.arange(
            1,
            260,
            dtype=np.int64,
        ),
    )
)


unit_cycle_unique_628hc7a5 = bool(

    not test_raw_628hc7a5
    .duplicated(
        [
            "unit_id",
            "cycle",
        ]
    )
    .any()
)


terminal_cycles_628hc7a5 = (

    test_raw_628hc7a5

    .groupby(
        "unit_id",
        sort=True,
    )[
        "cycle"
    ]

    .max()

    .to_numpy(
        dtype=np.int64
    )
)


print(
    "\n--- OFFICIAL FD002 TEST FEATURE CONTRACT ---"
)

print(
    "TEST path                    :",
    test_path_628hc7a5,
)

print(
    "Raw TEST shape               :",
    test_raw_628hc7a5.shape,
)

print(
    "Raw column count             :",
    test_raw_628hc7a5.shape[
        1
    ],
)

print(
    "TEST engines                 :",
    engine_count_628hc7a5,
)

print(
    "Min terminal observed cycle  :",
    int(
        terminal_cycles_628hc7a5.min()
    ),
)

print(
    "Median terminal obs. cycle   :",
    float(
        np.median(
            terminal_cycles_628hc7a5
        )
    ),
)

print(
    "Max terminal observed cycle  :",
    int(
        terminal_cycles_628hc7a5.max()
    ),
)

print(
    "Finite feature values        :",
    finite_ok_628hc7a5,
)

print(
    "Unique (unit_id, cycle) rows :",
    unit_cycle_unique_628hc7a5,
)


# --------------------------------------------------------------------------------------------------
# 5) RECOVER HISTORICAL FD002 PREPROCESSING HELPERS
#
# IMPORTANT:
# snapshot globals() first — avoids the earlier
# "dictionary changed size during iteration" failure mode.
# --------------------------------------------------------------------------------------------------

namespace_snapshot_628hc7a5 = list(
    globals().items()
)


callable_rows_628hc7a5 = []


for (
    name_628hc7a5,
    obj_628hc7a5,
) in namespace_snapshot_628hc7a5:

    lname_628hc7a5 = str(
        name_628hc7a5
    ).lower()


    relevant_628hc7a5 = (

        (
            "fd002"
            in
            lname_628hc7a5

            or

            "628"
            in
            lname_628hc7a5
        )

        and

        any(

            token_628hc7a5
            in
            lname_628hc7a5

            for token_628hc7a5 in (

                "normal",

                "window",

                "regime",

                "operat",

                "scale",

                "cluster",
            )
        )
    )


    if (
        relevant_628hc7a5
        and
        callable(
            obj_628hc7a5
        )
    ):

        try:

            signature_628hc7a5 = str(
                inspect.signature(
                    obj_628hc7a5
                )
            )

        except Exception:

            signature_628hc7a5 = (
                "<signature unavailable>"
            )


        callable_rows_628hc7a5.append(

            {
                "name":
                    str(
                        name_628hc7a5
                    ),

                "signature":
                    signature_628hc7a5,
            }
        )


if callable_rows_628hc7a5:

    FD002_C7A5_PREPROCESSING_CALLABLE_CANDIDATES_628HC7A5 = (

        pd.DataFrame(
            callable_rows_628hc7a5
        )

        .drop_duplicates(
            subset=[
                "name"
            ]
        )

        .sort_values(
            "name",
            kind="mergesort",
        )

        .reset_index(
            drop=True
        )
    )

else:

    FD002_C7A5_PREPROCESSING_CALLABLE_CANDIDATES_628HC7A5 = (

        pd.DataFrame(
            columns=[
                "name",
                "signature",
            ]
        )
    )


print(
    "\n--- FD002 PREPROCESSING / WINDOW CALLABLE CANDIDATES ---"
)

display(
    FD002_C7A5_PREPROCESSING_CALLABLE_CANDIDATES_628HC7A5
)


# --------------------------------------------------------------------------------------------------
# 6) RECOVER HISTORICAL REFERENCE / NORMALIZATION OBJECTS
# --------------------------------------------------------------------------------------------------

object_rows_628hc7a5 = []


for (
    name_628hc7a5,
    obj_628hc7a5,
) in namespace_snapshot_628hc7a5:

    lname_628hc7a5 = str(
        name_628hc7a5
    ).lower()


    relevant_628hc7a5 = (

        "fd002"
        in
        lname_628hc7a5

        and

        any(

            token_628hc7a5
            in
            lname_628hc7a5

            for token_628hc7a5 in (

                "fit",

                "normal",

                "window",

                "regime",

                "reference",

                "scaler",

                "cluster",

                "operat",
            )
        )
    )


    if not relevant_628hc7a5:

        continue


    if isinstance(
        obj_628hc7a5,
        pd.DataFrame,
    ):

        object_rows_628hc7a5.append(

            {
                "name":
                    str(
                        name_628hc7a5
                    ),

                "type":
                    "DataFrame",

                "shape_or_len":
                    str(
                        tuple(
                            obj_628hc7a5.shape
                        )
                    ),
            }
        )


    elif isinstance(
        obj_628hc7a5,
        np.ndarray,
    ):

        object_rows_628hc7a5.append(

            {
                "name":
                    str(
                        name_628hc7a5
                    ),

                "type":
                    "ndarray",

                "shape_or_len":
                    str(
                        tuple(
                            obj_628hc7a5.shape
                        )
                    ),
            }
        )


    elif isinstance(
        obj_628hc7a5,
        dict,
    ):

        object_rows_628hc7a5.append(

            {
                "name":
                    str(
                        name_628hc7a5
                    ),

                "type":
                    "dict",

                "shape_or_len":
                    f"dict[{len(obj_628hc7a5)}]",
            }
        )


if object_rows_628hc7a5:

    FD002_C7A5_PREPROCESSING_OBJECT_CANDIDATES_628HC7A5 = (

        pd.DataFrame(
            object_rows_628hc7a5
        )

        .drop_duplicates(
            subset=[
                "name"
            ]
        )

        .sort_values(
            "name",
            kind="mergesort",
        )

        .reset_index(
            drop=True
        )
    )

else:

    FD002_C7A5_PREPROCESSING_OBJECT_CANDIDATES_628HC7A5 = (

        pd.DataFrame(
            columns=[
                "name",
                "type",
                "shape_or_len",
            ]
        )
    )


print(
    "\n--- FD002 PREPROCESSING / REFERENCE OBJECT CANDIDATES ---"
)

display(
    FD002_C7A5_PREPROCESSING_OBJECT_CANDIDATES_628HC7A5
)


# --------------------------------------------------------------------------------------------------
# 7) KNOWN HISTORICAL HELPERS
# --------------------------------------------------------------------------------------------------

print(
    "\n--- KNOWN HISTORICAL FD002 HELPERS ---"
)


known_helpers_628hc7a5 = (

    "build_windows_628hc0",

    "normalize_corrupted_sensor_628ha",

    "apply_scenario_628hc0",
)


for helper_name_628hc7a5 in known_helpers_628hc7a5:

    helper_obj_628hc7a5 = globals().get(
        helper_name_628hc7a5
    )


    if callable(
        helper_obj_628hc7a5
    ):

        try:

            helper_sig_628hc7a5 = str(
                inspect.signature(
                    helper_obj_628hc7a5
                )
            )

        except Exception:

            helper_sig_628hc7a5 = (
                "<signature unavailable>"
            )


        print(

            f"{helper_name_628hc7a5:38s} : "
            f"PRESENT {helper_sig_628hc7a5}"
        )


    else:

        print(

            f"{helper_name_628hc7a5:38s} : "
            "absent"
        )


# --------------------------------------------------------------------------------------------------
# 8) AUDIT
# --------------------------------------------------------------------------------------------------

audit_628hc7a5 = pd.DataFrame(

    {
        "check": [

            "previous_C7_A4_passed",

            "test_feature_path_locked",

            "official_rul_path_still_locked_only",

            "raw_test_shape_33991x26",

            "raw_columns_exact_26",

            "raw_test_finite",

            "unit_ids_integer",

            "cycles_integer",

            "test_engine_count_259",

            "engine_ids_exact_1_to_259",

            "unit_cycle_pairs_unique",

            "test_features_read_once",

            "official_rul_contents_not_read",

            "test_not_scored",

            "no_model_execution",

            "no_mc_dropout_execution",

            "no_model_retraining",

            "no_dev_retuning",

            "no_threshold_or_rule_modification",

            "no_files_written",
        ],

        "passed": [

            True,

            True,

            True,

            raw_shape_ok_628hc7a5,

            raw_columns_ok_628hc7a5,

            finite_ok_628hc7a5,

            integer_ids_ok_628hc7a5,

            integer_cycles_ok_628hc7a5,

            engine_count_628hc7a5
            ==
            259,

            engine_sequence_ok_628hc7a5,

            unit_cycle_unique_628hc7a5,

            bool(
                FD002_OFFICIAL_TEST_FEATURES_READ_628HC7A5
            ),

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)


print(
    "\n--- C7-A5 AUDIT ---"
)

display(
    audit_628hc7a5
)


all_checks_passed_628hc7a5 = bool(

    audit_628hc7a5[
        "passed"
    ].all()
)


if not all_checks_passed_628hc7a5:

    raise RuntimeError(

        "C7-A5 TEST feature/input contract did not pass. "
        "Do not execute normalization, model inference, "
        "MC dropout, or open RUL_FD002.txt."
    )


# --------------------------------------------------------------------------------------------------
# 9) FINAL SUMMARY
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 126
)

print(
    "STEP 6.28H-C7-A5 FINAL SUMMARY"
)

print(
    "=" * 126
)

print(
    "Official FD002 TEST features read once :",
    True,
)

print(
    "Raw TEST shape                         :",
    test_raw_628hc7a5.shape,
)

print(
    "TEST engine count                      :",
    engine_count_628hc7a5,
)

print(
    "Raw contract valid                     :",
    True,
)

print(
    "Preprocessing callable candidates      :",
    len(
        FD002_C7A5_PREPROCESSING_CALLABLE_CANDIDATES_628HC7A5
    ),
)

print(
    "Preprocessing/reference candidates     :",
    len(
        FD002_C7A5_PREPROCESSING_OBJECT_CANDIDATES_628HC7A5
    ),
)

print(
    "Official RUL contents read             :",
    False,
)

print(
    "TEST scored                            :",
    False,
)

print(
    "Model/MC executed                      :",
    False,
)

print(
    "DEV retuning                           :",
    False,
)

print(
    "Threshold/rule modified                :",
    False,
)

print(
    "Files written                          :",
    False,
)

print(
    "All C7-A5 checks passed                :",
    all_checks_passed_628hc7a5,
)


print(
    "\nPASS — official FD002 TEST FEATURES are locked in memory. "
    "RUL_FD002.txt remains unopened."
)

print(
    "Send this output back. The next cell will bind the exact recovered "
    "FD002 normalization/window bridge and build the TEST input tensor "
    "without label access or policy retuning."
)

print(
    "\n=== END STEP 6.28H-C7-A5 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A6 — FD002 HISTORICAL NORMALIZED-TEST / ENDPOINT-WINDOW BRIDGE LOCK
#
# PURPOSE
# -------
# Lock the exact historical FD002 TEST-normalization object and inspect the exact historical
# endpoint-window builder + model-feature-list candidates already present in memory.
#
# IMPORTANT
# ---------
# - NO second TEST-file read
# - NO RUL_FD002.txt read
# - NO model execution
# - NO MC dropout
# - NO DEV retuning
# - NO threshold/rule modification
# - NO files written
# ==================================================================================================

import inspect
import numpy as np
import pandas as pd

print("=" * 126)
print("STEP 6.28H-C7-A6 — FD002 HISTORICAL NORMALIZED-TEST / ENDPOINT-WINDOW BRIDGE LOCK")
print("=" * 126)


# --------------------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a5", False)):
    raise RuntimeError("C7-A5 must pass before C7-A6.")

if "FD002_OFFICIAL_TEST_RAW_628HC7A5" not in globals():
    raise RuntimeError("C7-A5 raw FD002 TEST object is missing from memory.")

if "FD002_TEST_NORMALIZED_628C" not in globals():
    raise RuntimeError("Historical FD002_TEST_NORMALIZED_628C object is missing.")

if "build_test_endpoint_windows_628d" not in globals():
    raise RuntimeError("Historical build_test_endpoint_windows_628d helper is missing.")


raw_test_628hc7a6 = FD002_OFFICIAL_TEST_RAW_628HC7A5
normalized_test_628hc7a6 = FD002_TEST_NORMALIZED_628C
endpoint_builder_628hc7a6 = build_test_endpoint_windows_628d


if not isinstance(raw_test_628hc7a6, pd.DataFrame):
    raise RuntimeError("C7-A5 raw TEST object is not a DataFrame.")

if not isinstance(normalized_test_628hc7a6, pd.DataFrame):
    raise RuntimeError("FD002_TEST_NORMALIZED_628C is not a DataFrame.")


# --------------------------------------------------------------------------------------------------
# 2) Historical normalized TEST identity
# --------------------------------------------------------------------------------------------------

identifier_columns_628hc7a6 = [
    "unit_id",
    "cycle",
]

identifiers_present_628hc7a6 = bool(
    all(
        c in normalized_test_628hc7a6.columns
        for c in identifier_columns_628hc7a6
    )
)

if not identifiers_present_628hc7a6:
    raise RuntimeError(
        "Historical normalized TEST object does not contain "
        "the required unit_id/cycle identifiers."
    )


same_row_count_628hc7a6 = bool(
    len(normalized_test_628hc7a6)
    ==
    len(raw_test_628hc7a6)
    ==
    33991
)


unit_identity_628hc7a6 = bool(
    np.array_equal(
        normalized_test_628hc7a6["unit_id"].to_numpy(),
        raw_test_628hc7a6["unit_id"].to_numpy(),
    )
)


cycle_identity_628hc7a6 = bool(
    np.array_equal(
        normalized_test_628hc7a6["cycle"].to_numpy(),
        raw_test_628hc7a6["cycle"].to_numpy(),
    )
)


numeric_norm_628hc7a6 = (
    normalized_test_628hc7a6
    .select_dtypes(include=[np.number])
)


normalized_numeric_finite_628hc7a6 = bool(
    np.isfinite(
        numeric_norm_628hc7a6.to_numpy(dtype=np.float64)
    ).all()
)


target_tokens_628hc7a6 = (
    "rul",
    "target",
    "label",
    "truth",
    "remaining_useful",
)


target_like_columns_628hc7a6 = [
    str(c)
    for c in normalized_test_628hc7a6.columns
    if any(
        token in str(c).lower()
        for token in target_tokens_628hc7a6
    )
]


no_target_like_columns_628hc7a6 = bool(
    len(target_like_columns_628hc7a6) == 0
)


print("\n--- HISTORICAL NORMALIZED TEST OBJECT ---")

print(
    "Object name                 :",
    "FD002_TEST_NORMALIZED_628C",
)

print(
    "Shape                       :",
    normalized_test_628hc7a6.shape,
)

print(
    "Raw/normalized row identity :",
    same_row_count_628hc7a6,
)

print(
    "unit_id identity            :",
    unit_identity_628hc7a6,
)

print(
    "cycle identity              :",
    cycle_identity_628hc7a6,
)

print(
    "Numeric values finite       :",
    normalized_numeric_finite_628hc7a6,
)

print(
    "Target-like columns         :",
    target_like_columns_628hc7a6,
)


print("\nNormalized TEST columns:")

print(
    list(
        normalized_test_628hc7a6.columns
    )
)


# --------------------------------------------------------------------------------------------------
# 3) Exact historical endpoint-window builder contract
# --------------------------------------------------------------------------------------------------

print("\n--- HISTORICAL ENDPOINT-WINDOW BUILDER ---")

print(
    "name      :",
    "build_test_endpoint_windows_628d",
)

print(
    "signature :",
    inspect.signature(
        endpoint_builder_628hc7a6
    ),
)


try:

    endpoint_builder_source_628hc7a6 = (
        inspect.getsource(
            endpoint_builder_628hc7a6
        )
    )

except Exception as exc:

    endpoint_builder_source_628hc7a6 = (
        f"<source unavailable: "
        f"{type(exc).__name__}: {exc}>"
    )


print("\n--- BUILDER SOURCE ---")

print(
    endpoint_builder_source_628hc7a6
)


# --------------------------------------------------------------------------------------------------
# 4) Discover exact 17-feature-list candidates already present in memory
#
# Snapshot globals first so namespace inspection cannot mutate during iteration.
# --------------------------------------------------------------------------------------------------

namespace_snapshot_628hc7a6 = list(
    globals().items()
)


normalized_columns_set_628hc7a6 = set(
    map(
        str,
        normalized_test_628hc7a6.columns,
    )
)


feature_alias_rows_628hc7a6 = []


for (
    name_628hc7a6,
    obj_628hc7a6,
) in namespace_snapshot_628hc7a6:

    values_628hc7a6 = None


    if isinstance(
        obj_628hc7a6,
        (
            list,
            tuple,
            pd.Index,
        ),
    ):

        values_628hc7a6 = list(
            obj_628hc7a6
        )


    elif (
        isinstance(
            obj_628hc7a6,
            np.ndarray,
        )
        and
        obj_628hc7a6.ndim == 1
    ):

        values_628hc7a6 = (
            obj_628hc7a6.tolist()
        )


    if (
        values_628hc7a6 is None
        or
        len(values_628hc7a6) != 17
    ):
        continue


    if not all(
        isinstance(v, str)
        for v in values_628hc7a6
    ):
        continue


    values_628hc7a6 = [
        str(v)
        for v in values_628hc7a6
    ]


    if not set(
        values_628hc7a6
    ).issubset(
        normalized_columns_set_628hc7a6
    ):
        continue


    feature_alias_rows_628hc7a6.append(
        {
            "alias_name":
                str(
                    name_628hc7a6
                ),

            "feature_signature":
                tuple(
                    values_628hc7a6
                ),
        }
    )


FD002_C7A6_FEATURE_LIST_ALIASES_628HC7A6 = pd.DataFrame(
    feature_alias_rows_628hc7a6
)


if len(
    FD002_C7A6_FEATURE_LIST_ALIASES_628HC7A6
) > 0:

    unique_signatures_628hc7a6 = list(
        dict.fromkeys(
            FD002_C7A6_FEATURE_LIST_ALIASES_628HC7A6[
                "feature_signature"
            ].tolist()
        )
    )

else:

    unique_signatures_628hc7a6 = []


print("\n--- 17-FEATURE LIST ALIASES ---")


if len(
    FD002_C7A6_FEATURE_LIST_ALIASES_628HC7A6
) == 0:

    print(
        "No in-memory 17-column string-list "
        "candidate was found."
    )

else:

    display(
        FD002_C7A6_FEATURE_LIST_ALIASES_628HC7A6
    )


print(
    "\nUnique 17-feature signatures :",
    len(
        unique_signatures_628hc7a6
    ),
)


for (
    idx_628hc7a6,
    sig_628hc7a6,
) in enumerate(
    unique_signatures_628hc7a6,
    start=1,
):

    print(
        f"\nSignature {idx_628hc7a6}:"
    )

    print(
        list(
            sig_628hc7a6
        )
    )


# --------------------------------------------------------------------------------------------------
# 5) Discover historical endpoint tensor candidates, if any
# --------------------------------------------------------------------------------------------------

endpoint_tensor_rows_628hc7a6 = []


for (
    name_628hc7a6,
    obj_628hc7a6,
) in namespace_snapshot_628hc7a6:

    if (
        isinstance(
            obj_628hc7a6,
            np.ndarray,
        )
        and
        obj_628hc7a6.shape
        ==
        (
            259,
            30,
            17,
        )
    ):

        endpoint_tensor_rows_628hc7a6.append(
            {
                "name":
                    str(
                        name_628hc7a6
                    ),

                "dtype":
                    str(
                        obj_628hc7a6.dtype
                    ),

                "finite":
                    bool(
                        np.isfinite(
                            obj_628hc7a6
                        ).all()
                    ),
            }
        )


FD002_C7A6_HISTORICAL_ENDPOINT_TENSOR_CANDIDATES_628HC7A6 = pd.DataFrame(
    endpoint_tensor_rows_628hc7a6
)


print(
    "\n--- HISTORICAL (259, 30, 17) "
    "ENDPOINT-TENSOR CANDIDATES ---"
)


if len(
    FD002_C7A6_HISTORICAL_ENDPOINT_TENSOR_CANDIDATES_628HC7A6
) == 0:

    print(
        "No historical (259, 30, 17) ndarray "
        "is currently present."
    )

else:

    display(
        FD002_C7A6_HISTORICAL_ENDPOINT_TENSOR_CANDIDATES_628HC7A6
    )


# --------------------------------------------------------------------------------------------------
# 6) Lock objects for next step
# --------------------------------------------------------------------------------------------------

FD002_C7A6_LOCKED_NORMALIZED_TEST_OBJECT_628HC7A6 = (
    "FD002_TEST_NORMALIZED_628C"
)


FD002_C7A6_LOCKED_ENDPOINT_BUILDER_628HC7A6 = (
    "build_test_endpoint_windows_628d"
)


FD002_C7A6_UNIQUE_FEATURE_SIGNATURES_628HC7A6 = tuple(
    unique_signatures_628hc7a6
)


# --------------------------------------------------------------------------------------------------
# 7) Audit
# --------------------------------------------------------------------------------------------------

audit_628hc7a6 = pd.DataFrame(
    {
        "check": [

            "previous_C7_A5_passed",

            "historical_normalized_test_shape_33991_rows",

            "unit_id_identity_with_one_time_raw_read",

            "cycle_identity_with_one_time_raw_read",

            "normalized_numeric_values_finite",

            "normalized_test_has_no_target_like_columns",

            "endpoint_builder_present_callable",

            "feature_signature_candidate_found",

            "official_rul_contents_not_read",

            "test_not_scored",

            "no_model_execution",

            "no_mc_dropout_execution",

            "no_model_retraining",

            "no_dev_retuning",

            "no_threshold_or_rule_modification",

            "no_additional_test_file_read",

            "no_files_written",
        ],

        "passed": [

            True,

            same_row_count_628hc7a6,

            unit_identity_628hc7a6,

            cycle_identity_628hc7a6,

            normalized_numeric_finite_628hc7a6,

            no_target_like_columns_628hc7a6,

            callable(
                endpoint_builder_628hc7a6
            ),

            len(
                unique_signatures_628hc7a6
            ) >= 1,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)


print(
    "\n--- C7-A6 AUDIT ---"
)

display(
    audit_628hc7a6
)


all_checks_passed_628hc7a6 = bool(
    audit_628hc7a6[
        "passed"
    ].all()
)


if not all_checks_passed_628hc7a6:

    raise RuntimeError(
        "C7-A6 bridge-lock audit did not pass. "
        "Do not build TEST windows or execute the model."
    )


# --------------------------------------------------------------------------------------------------
# 8) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 126
)

print(
    "STEP 6.28H-C7-A6 FINAL SUMMARY"
)

print(
    "=" * 126
)


print(
    "Historical normalized TEST locked   :",
    FD002_C7A6_LOCKED_NORMALIZED_TEST_OBJECT_628HC7A6,
)

print(
    "Historical endpoint builder locked  :",
    FD002_C7A6_LOCKED_ENDPOINT_BUILDER_628HC7A6,
)

print(
    "Normalized TEST shape               :",
    normalized_test_628hc7a6.shape,
)

print(
    "Raw/normalized identifiers exact    :",
    (
        unit_identity_628hc7a6
        and
        cycle_identity_628hc7a6
    ),
)

print(
    "Unique 17-feature signatures found  :",
    len(
        unique_signatures_628hc7a6
    ),
)

print(
    "Historical endpoint tensors found   :",
    len(
        FD002_C7A6_HISTORICAL_ENDPOINT_TENSOR_CANDIDATES_628HC7A6
    ),
)

print(
    "Official RUL contents read          :",
    False,
)

print(
    "TEST scored                          :",
    False,
)

print(
    "Model/MC executed                    :",
    False,
)

print(
    "DEV retuning                         :",
    False,
)

print(
    "Threshold/rule modified              :",
    False,
)

print(
    "Additional TEST-file read            :",
    False,
)

print(
    "Files written                        :",
    False,
)

print(
    "All C7-A6 checks passed              :",
    all_checks_passed_628hc7a6,
)


print(
    "\nPASS — the historical FD002 normalized TEST representation "
    "and exact endpoint-window builder are now provenance-locked "
    "without touching RUL_FD002.txt."
)

print(
    "Send this output back. The next cell will resolve the exact "
    "17-feature ordering, build the 259 endpoint windows, and verify "
    "them against any historical endpoint tensor before any model "
    "or MC-dropout execution."
)

print(
    "\n=== END STEP 6.28H-C7-A6 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A7 — FD002 EXACT 17-FEATURE ORDER + ENDPOINT-TENSOR IDENTITY LOCK
#
# PURPOSE
# -------
# Resolve the exact historical 17-feature ordering by rebuilding the FD002 TEST endpoint tensor
# under every recovered 17-feature signature and requiring an EXACT match to the canonical
# historical tensor X_TEST_FD002_628D.
#
# SAFETY
# ------
# - Uses the already-read TEST features / historical normalized TEST object only
# - Does NOT read RUL_FD002.txt
# - Does NOT execute the model
# - Does NOT execute MC dropout
# - Does NOT score TEST
# - Does NOT retune DEV / threshold / policy
# - Writes no files
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 126)
print("STEP 6.28H-C7-A7 — FD002 EXACT 17-FEATURE ORDER + ENDPOINT-TENSOR IDENTITY LOCK")
print("=" * 126)


# --------------------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a6", False)):
    raise RuntimeError("C7-A6 must pass before C7-A7.")

required_names_628hc7a7 = [
    "FD002_TEST_NORMALIZED_628C",
    "build_test_endpoint_windows_628d",
    "FD002_C7A6_UNIQUE_FEATURE_SIGNATURES_628HC7A6",
    "X_TEST_FD002_628D",
]

missing_628hc7a7 = [
    name
    for name in required_names_628hc7a7
    if name not in globals()
]

if missing_628hc7a7:
    raise RuntimeError(
        "Missing required C7-A7 objects: "
        + ", ".join(missing_628hc7a7)
    )

normalized_test_628hc7a7 = FD002_TEST_NORMALIZED_628C
builder_628hc7a7 = build_test_endpoint_windows_628d

feature_signatures_628hc7a7 = tuple(
    FD002_C7A6_UNIQUE_FEATURE_SIGNATURES_628HC7A6
)

historical_x_628hc7a7 = np.asarray(
    X_TEST_FD002_628D
)

if historical_x_628hc7a7.shape != (259, 30, 17):
    raise RuntimeError(
        "Canonical historical X_TEST_FD002_628D has unexpected shape: "
        f"{historical_x_628hc7a7.shape}"
    )

if not np.isfinite(historical_x_628hc7a7).all():
    raise RuntimeError(
        "Canonical historical X_TEST_FD002_628D contains non-finite values."
    )

if len(feature_signatures_628hc7a7) < 1:
    raise RuntimeError(
        "No recovered 17-feature signatures are available."
    )


# --------------------------------------------------------------------------------------------------
# 2) Verify historical endpoint-tensor aliases are identical to canonical tensor
# --------------------------------------------------------------------------------------------------

historical_alias_names_628hc7a7 = [
    "X_TEST_FD002_628D",
    "X_test_628fa",
    "X_test_628far2",
    "X_test_628fbr1",
]

alias_rows_628hc7a7 = []

for alias_name_628hc7a7 in historical_alias_names_628hc7a7:

    if alias_name_628hc7a7 not in globals():

        alias_rows_628hc7a7.append(
            {
                "name": alias_name_628hc7a7,
                "present": False,
                "shape": None,
                "finite": None,
                "exact_to_canonical": None,
                "max_abs_difference": np.nan,
            }
        )

        continue

    alias_arr_628hc7a7 = np.asarray(
        globals()[alias_name_628hc7a7]
    )

    same_shape_628hc7a7 = bool(
        alias_arr_628hc7a7.shape
        ==
        historical_x_628hc7a7.shape
    )

    finite_alias_628hc7a7 = bool(
        np.isfinite(
            alias_arr_628hc7a7
        ).all()
    )

    exact_alias_628hc7a7 = bool(
        same_shape_628hc7a7
        and
        np.array_equal(
            alias_arr_628hc7a7,
            historical_x_628hc7a7,
        )
    )

    max_abs_alias_628hc7a7 = (
        float(
            np.max(
                np.abs(
                    alias_arr_628hc7a7.astype(np.float64)
                    -
                    historical_x_628hc7a7.astype(np.float64)
                )
            )
        )
        if same_shape_628hc7a7
        else np.nan
    )

    alias_rows_628hc7a7.append(
        {
            "name": alias_name_628hc7a7,
            "present": True,
            "shape": alias_arr_628hc7a7.shape,
            "finite": finite_alias_628hc7a7,
            "exact_to_canonical": exact_alias_628hc7a7,
            "max_abs_difference": max_abs_alias_628hc7a7,
        }
    )


FD002_C7A7_HISTORICAL_TENSOR_ALIAS_AUDIT_628HC7A7 = pd.DataFrame(
    alias_rows_628hc7a7
)

print("\n--- HISTORICAL ENDPOINT-TENSOR ALIAS IDENTITY ---")

display(
    FD002_C7A7_HISTORICAL_TENSOR_ALIAS_AUDIT_628HC7A7
)

present_alias_rows_628hc7a7 = (
    FD002_C7A7_HISTORICAL_TENSOR_ALIAS_AUDIT_628HC7A7[
        FD002_C7A7_HISTORICAL_TENSOR_ALIAS_AUDIT_628HC7A7[
            "present"
        ]
    ]
)

historical_aliases_exact_628hc7a7 = bool(
    len(
        present_alias_rows_628hc7a7
    ) >= 1
    and
    present_alias_rows_628hc7a7[
        "exact_to_canonical"
    ].fillna(False).all()
)


# --------------------------------------------------------------------------------------------------
# 3) Rebuild endpoint tensor under EVERY recovered feature signature
# --------------------------------------------------------------------------------------------------

signature_rows_628hc7a7 = []

rebuilt_by_signature_628hc7a7 = {}
padding_by_signature_628hc7a7 = {}
metadata_by_signature_628hc7a7 = {}

for (
    signature_index_628hc7a7,
    signature_628hc7a7,
) in enumerate(
    feature_signatures_628hc7a7,
    start=1,
):

    feature_list_628hc7a7 = list(
        signature_628hc7a7
    )

    missing_features_628hc7a7 = [
        c
        for c in feature_list_628hc7a7
        if c not in normalized_test_628hc7a7.columns
    ]

    if missing_features_628hc7a7:
        raise RuntimeError(
            f"Signature {signature_index_628hc7a7} contains missing columns: "
            f"{missing_features_628hc7a7}"
        )

    (
        rebuilt_x_628hc7a7,
        padding_mask_628hc7a7,
        endpoint_metadata_628hc7a7,
    ) = builder_628hc7a7(
        normalized_test_628hc7a7,
        feature_list_628hc7a7,
        30,
    )

    rebuilt_x_628hc7a7 = np.asarray(
        rebuilt_x_628hc7a7
    )

    padding_mask_628hc7a7 = np.asarray(
        padding_mask_628hc7a7
    )

    if rebuilt_x_628hc7a7.shape != (259, 30, 17):
        raise RuntimeError(
            f"Signature {signature_index_628hc7a7} rebuilt unexpected shape "
            f"{rebuilt_x_628hc7a7.shape}."
        )

    if padding_mask_628hc7a7.shape != (259, 30):
        raise RuntimeError(
            f"Signature {signature_index_628hc7a7} padding mask unexpected shape "
            f"{padding_mask_628hc7a7.shape}."
        )

    if not np.isfinite(
        rebuilt_x_628hc7a7
    ).all():

        raise RuntimeError(
            f"Signature {signature_index_628hc7a7} produced non-finite windows."
        )

    exact_match_628hc7a7 = bool(
        np.array_equal(
            rebuilt_x_628hc7a7,
            historical_x_628hc7a7,
        )
    )

    max_abs_difference_628hc7a7 = float(
        np.max(
            np.abs(
                rebuilt_x_628hc7a7.astype(np.float64)
                -
                historical_x_628hc7a7.astype(np.float64)
            )
        )
    )

    mean_abs_difference_628hc7a7 = float(
        np.mean(
            np.abs(
                rebuilt_x_628hc7a7.astype(np.float64)
                -
                historical_x_628hc7a7.astype(np.float64)
            )
        )
    )

    rebuilt_by_signature_628hc7a7[
        signature_index_628hc7a7
    ] = rebuilt_x_628hc7a7

    padding_by_signature_628hc7a7[
        signature_index_628hc7a7
    ] = padding_mask_628hc7a7

    metadata_by_signature_628hc7a7[
        signature_index_628hc7a7
    ] = endpoint_metadata_628hc7a7.copy()

    signature_rows_628hc7a7.append(
        {
            "signature_index":
                signature_index_628hc7a7,

            "feature_signature":
                tuple(
                    feature_list_628hc7a7
                ),

            "tensor_shape":
                rebuilt_x_628hc7a7.shape,

            "finite":
                bool(
                    np.isfinite(
                        rebuilt_x_628hc7a7
                    ).all()
                ),

            "exact_match_historical":
                exact_match_628hc7a7,

            "max_abs_difference":
                max_abs_difference_628hc7a7,

            "mean_abs_difference":
                mean_abs_difference_628hc7a7,

            "padded_engines":
                int(
                    endpoint_metadata_628hc7a7[
                        "padding_used"
                    ].sum()
                ),

            "max_padding_count":
                int(
                    endpoint_metadata_628hc7a7[
                        "padding_count"
                    ].max()
                ),
        }
    )


FD002_C7A7_FEATURE_SIGNATURE_IDENTITY_628HC7A7 = pd.DataFrame(
    signature_rows_628hc7a7
)

print(
    "\n--- FEATURE-SIGNATURE -> HISTORICAL TENSOR IDENTITY ---"
)

display(
    FD002_C7A7_FEATURE_SIGNATURE_IDENTITY_628HC7A7
)


# --------------------------------------------------------------------------------------------------
# 4) Require ONE AND ONLY ONE exact feature-signature match
# --------------------------------------------------------------------------------------------------

exact_signature_rows_628hc7a7 = (
    FD002_C7A7_FEATURE_SIGNATURE_IDENTITY_628HC7A7[
        FD002_C7A7_FEATURE_SIGNATURE_IDENTITY_628HC7A7[
            "exact_match_historical"
        ]
    ]
)

if len(
    exact_signature_rows_628hc7a7
) != 1:

    raise RuntimeError(
        "Expected exactly one 17-feature signature to reproduce "
        "X_TEST_FD002_628D exactly; found "
        f"{len(exact_signature_rows_628hc7a7)}. "
        "Do not execute the model."
    )


selected_signature_index_628hc7a7 = int(
    exact_signature_rows_628hc7a7.iloc[0][
        "signature_index"
    ]
)

selected_feature_signature_628hc7a7 = tuple(
    feature_signatures_628hc7a7[
        selected_signature_index_628hc7a7
        -
        1
    ]
)

selected_feature_list_628hc7a7 = list(
    selected_feature_signature_628hc7a7
)

selected_x_628hc7a7 = np.asarray(
    rebuilt_by_signature_628hc7a7[
        selected_signature_index_628hc7a7
    ]
)

selected_padding_mask_628hc7a7 = np.asarray(
    padding_by_signature_628hc7a7[
        selected_signature_index_628hc7a7
    ]
)

selected_endpoint_metadata_628hc7a7 = (
    metadata_by_signature_628hc7a7[
        selected_signature_index_628hc7a7
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 5) Cross-check against named FD002 feature list
# --------------------------------------------------------------------------------------------------

fd002_named_feature_list_present_628hc7a7 = bool(
    "FD002_MODEL_FEATURES_628C"
    in globals()
)

fd002_named_feature_list_exact_628hc7a7 = None

if fd002_named_feature_list_present_628hc7a7:

    fd002_named_feature_list_exact_628hc7a7 = bool(
        tuple(
            list(
                FD002_MODEL_FEATURES_628C
            )
        )
        ==
        selected_feature_signature_628hc7a7
    )

    if not fd002_named_feature_list_exact_628hc7a7:

        raise RuntimeError(
            "Exact tensor identity selected a feature signature that does not "
            "match FD002_MODEL_FEATURES_628C. Do not execute the model."
        )


# --------------------------------------------------------------------------------------------------
# 6) Endpoint metadata contract
# --------------------------------------------------------------------------------------------------

expected_unit_ids_628hc7a7 = np.arange(
    1,
    260,
    dtype=np.int64,
)

endpoint_units_exact_628hc7a7 = bool(
    np.array_equal(
        selected_endpoint_metadata_628hc7a7[
            "unit_id"
        ].to_numpy(
            dtype=np.int64
        ),
        expected_unit_ids_628hc7a7,
    )
)

terminal_cycles_from_raw_628hc7a7 = (
    FD002_OFFICIAL_TEST_RAW_628HC7A5
    .groupby(
        "unit_id",
        sort=True,
    )["cycle"]
    .max()
    .to_numpy(
        dtype=np.int64
    )
)

endpoint_terminal_cycles_exact_628hc7a7 = bool(
    np.array_equal(
        selected_endpoint_metadata_628hc7a7[
            "cycle"
        ].to_numpy(
            dtype=np.int64
        ),
        terminal_cycles_from_raw_628hc7a7,
    )
)

endpoint_tensor_exact_628hc7a7 = bool(
    np.array_equal(
        selected_x_628hc7a7,
        historical_x_628hc7a7,
    )
)


# --------------------------------------------------------------------------------------------------
# 7) Freeze in-memory TEST execution inputs
# --------------------------------------------------------------------------------------------------

FD002_C7A7_LOCKED_MODEL_FEATURES_628HC7A7 = tuple(
    selected_feature_signature_628hc7a7
)

FD002_C7A7_LOCKED_TEST_ENDPOINT_WINDOWS_628HC7A7 = (
    selected_x_628hc7a7.copy()
)

FD002_C7A7_LOCKED_TEST_PADDING_MASK_628HC7A7 = (
    selected_padding_mask_628hc7a7.copy()
)

FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7 = (
    selected_endpoint_metadata_628hc7a7.copy()
)


# --------------------------------------------------------------------------------------------------
# 8) Audit
# --------------------------------------------------------------------------------------------------

audit_628hc7a7 = pd.DataFrame(
    {
        "check": [

            "previous_C7_A6_passed",

            "historical_tensor_shape_259x30x17",

            "historical_tensor_finite",

            "historical_tensor_aliases_exact",

            "exactly_one_feature_signature_matches",

            "selected_tensor_exact_historical",

            "selected_endpoint_units_exact_1_to_259",

            "selected_terminal_cycles_exact",

            "named_FD002_feature_list_consistent_if_present",

            "official_rul_contents_not_read",

            "test_not_scored",

            "no_model_execution",

            "no_mc_dropout_execution",

            "no_model_retraining",

            "no_dev_retuning",

            "no_threshold_or_rule_modification",

            "no_additional_test_file_read",

            "no_files_written",
        ],

        "passed": [

            True,

            bool(
                historical_x_628hc7a7.shape
                ==
                (259, 30, 17)
            ),

            bool(
                np.isfinite(
                    historical_x_628hc7a7
                ).all()
            ),

            historical_aliases_exact_628hc7a7,

            bool(
                len(
                    exact_signature_rows_628hc7a7
                )
                ==
                1
            ),

            endpoint_tensor_exact_628hc7a7,

            endpoint_units_exact_628hc7a7,

            endpoint_terminal_cycles_exact_628hc7a7,

            (
                True
                if not fd002_named_feature_list_present_628hc7a7
                else bool(
                    fd002_named_feature_list_exact_628hc7a7
                )
            ),

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)

print(
    "\n--- C7-A7 AUDIT ---"
)

display(
    audit_628hc7a7
)

all_checks_passed_628hc7a7 = bool(
    audit_628hc7a7[
        "passed"
    ].all()
)

if not all_checks_passed_628hc7a7:

    raise RuntimeError(
        "C7-A7 did not pass. "
        "Do not execute FD002 model/MC dropout."
    )


# --------------------------------------------------------------------------------------------------
# 9) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 126
)

print(
    "STEP 6.28H-C7-A7 FINAL SUMMARY"
)

print(
    "=" * 126
)

print(
    "Exact historical feature signature selected :",
    selected_signature_index_628hc7a7,
)

print(
    "Locked model features                       :",
    list(
        FD002_C7A7_LOCKED_MODEL_FEATURES_628HC7A7
    ),
)

print(
    "Endpoint tensor shape                       :",
    FD002_C7A7_LOCKED_TEST_ENDPOINT_WINDOWS_628HC7A7.shape,
)

print(
    "Exact to X_TEST_FD002_628D                  :",
    endpoint_tensor_exact_628hc7a7,
)

print(
    "Historical aliases exact                    :",
    historical_aliases_exact_628hc7a7,
)

print(
    "Endpoint unit order exact 1..259            :",
    endpoint_units_exact_628hc7a7,
)

print(
    "Terminal cycles exact                       :",
    endpoint_terminal_cycles_exact_628hc7a7,
)

print(
    "Padded engines                              :",
    int(
        FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7[
            "padding_used"
        ].sum()
    ),
)

print(
    "Maximum left-padding count                  :",
    int(
        FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7[
            "padding_count"
        ].max()
    ),
)

print(
    "Official RUL contents read                  :",
    False,
)

print(
    "TEST scored                                  :",
    False,
)

print(
    "Model/MC executed                            :",
    False,
)

print(
    "DEV retuning                                 :",
    False,
)

print(
    "Threshold/rule modified                      :",
    False,
)

print(
    "Additional TEST-file read                    :",
    False,
)

print(
    "Files written                                :",
    False,
)

print(
    "All C7-A7 checks passed                      :",
    all_checks_passed_628hc7a7,
)

print(
    "\nPASS — the exact FD002 17-feature ordering and the 259-engine "
    "TEST endpoint tensor are now locked by exact historical identity."
)

print(
    "Next: recover and verify the frozen FD002 model/MC execution objects "
    "and pre-label prediction contract before any official RUL read."
)

print(
    "\n=== END STEP 6.28H-C7-A7 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A8 — FD002 FROZEN MODEL / MC HELPER + PRE-LABEL OUTPUT CONTRACT RECOVERY
#
# PURPOSE
# -------
# Recover the exact in-memory model, MC-dropout helper, historical execution references,
# and pre-label endpoint-output candidates needed for the one-way FD002 TEST execution.
#
# THIS STEP IS DISCOVERY / VERIFICATION ONLY.
#
# SAFETY
# ------
# - Uses the already-locked FD002 TEST endpoint tensor only as metadata/identity input.
# - Does NOT run the model.
# - Does NOT run MC dropout.
# - Does NOT open RUL_FD002.txt.
# - Does NOT score TEST.
# - Does NOT use TEST labels/targets/errors.
# - Does NOT retune DEV, the reliability rule, or the threshold.
# - Writes no files.
# ==================================================================================================

import inspect
import hashlib
import numpy as np
import pandas as pd
import torch

print("=" * 126)
print("STEP 6.28H-C7-A8 — FD002 FROZEN MODEL / MC HELPER + PRE-LABEL OUTPUT CONTRACT RECOVERY")
print("=" * 126)


# --------------------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a7", False)):
    raise RuntimeError("C7-A7 must pass before C7-A8.")

locked_x_628hc7a8 = np.asarray(
    FD002_C7A7_LOCKED_TEST_ENDPOINT_WINDOWS_628HC7A7
)

if locked_x_628hc7a8.shape != (259, 30, 17):
    raise RuntimeError(
        "Locked FD002 TEST endpoint tensor no longer has shape (259, 30, 17)."
    )

if not np.isfinite(locked_x_628hc7a8).all():
    raise RuntimeError(
        "Locked FD002 TEST endpoint tensor contains non-finite values."
    )

if not np.array_equal(
    locked_x_628hc7a8,
    np.asarray(X_TEST_FD002_628D),
):
    raise RuntimeError(
        "Locked FD002 TEST endpoint tensor no longer matches X_TEST_FD002_628D exactly."
    )

# Snapshot once — avoids 'dictionary changed size during iteration'.
global_items_628hc7a8 = list(globals().items())


# --------------------------------------------------------------------------------------------------
# 2) Torch model candidates — metadata only, NO forward pass
# --------------------------------------------------------------------------------------------------

model_rows_628hc7a8 = []

for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    if not isinstance(obj_628hc7a8, torch.nn.Module):
        continue

    try:
        parameter_count_628hc7a8 = int(
            sum(p.numel() for p in obj_628hc7a8.parameters())
        )
    except Exception:
        parameter_count_628hc7a8 = None

    try:
        trainable_parameter_count_628hc7a8 = int(
            sum(
                p.numel()
                for p in obj_628hc7a8.parameters()
                if p.requires_grad
            )
        )
    except Exception:
        trainable_parameter_count_628hc7a8 = None

    try:
        first_parameter_628hc7a8 = next(obj_628hc7a8.parameters())

        device_628hc7a8 = str(
            first_parameter_628hc7a8.device
        )

        dtype_628hc7a8 = str(
            first_parameter_628hc7a8.dtype
        )

    except Exception:
        device_628hc7a8 = None
        dtype_628hc7a8 = None

    input_size_hints_628hc7a8 = []

    try:

        for module_628hc7a8 in obj_628hc7a8.modules():

            if module_628hc7a8 is obj_628hc7a8:
                continue

            if hasattr(module_628hc7a8, "input_size"):

                try:
                    input_size_hints_628hc7a8.append(
                        int(module_628hc7a8.input_size)
                    )
                except Exception:
                    pass

            if isinstance(
                module_628hc7a8,
                torch.nn.Linear,
            ):

                input_size_hints_628hc7a8.append(
                    int(module_628hc7a8.in_features)
                )

            if len(input_size_hints_628hc7a8) >= 6:
                break

    except Exception:
        pass

    model_rows_628hc7a8.append(
        {
            "name": name_628hc7a8,
            "class": obj_628hc7a8.__class__.__name__,
            "parameter_count": parameter_count_628hc7a8,
            "trainable_parameters": trainable_parameter_count_628hc7a8,
            "training_mode": bool(obj_628hc7a8.training),
            "device": device_628hc7a8,
            "dtype": dtype_628hc7a8,
            "input_size_hints": tuple(
                input_size_hints_628hc7a8
            ),
        }
    )


FD002_C7A8_MODEL_CANDIDATES_628HC7A8 = pd.DataFrame(
    model_rows_628hc7a8
)

if len(FD002_C7A8_MODEL_CANDIDATES_628HC7A8) > 0:

    FD002_C7A8_MODEL_CANDIDATES_628HC7A8 = (
        FD002_C7A8_MODEL_CANDIDATES_628HC7A8
        .assign(
            fd002_name_hint=lambda d:
                d["name"]
                .astype(str)
                .str.lower()
                .str.contains("fd002")
        )
        .sort_values(
            ["fd002_name_hint", "name"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


print("\n--- TORCH MODEL CANDIDATES: METADATA ONLY ---")

if len(FD002_C7A8_MODEL_CANDIDATES_628HC7A8) == 0:

    print(
        "No torch.nn.Module objects found in current notebook state."
    )

else:

    display(
        FD002_C7A8_MODEL_CANDIDATES_628HC7A8
    )


# --------------------------------------------------------------------------------------------------
# 3) MC-dropout / prediction helper candidates — signatures only, NO execution
# --------------------------------------------------------------------------------------------------

helper_rows_628hc7a8 = []

for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    helper_name_match_628hc7a8 = bool(
        (
            "mc" in name_lower_628hc7a8
            and
            "dropout" in name_lower_628hc7a8
        )
        or
        (
            "mc" in name_lower_628hc7a8
            and
            "predict" in name_lower_628hc7a8
        )
    )

    if not (
        callable(obj_628hc7a8)
        and
        helper_name_match_628hc7a8
    ):
        continue

    try:

        signature_628hc7a8 = str(
            inspect.signature(
                obj_628hc7a8
            )
        )

    except Exception:

        signature_628hc7a8 = (
            "<signature unavailable>"
        )

    helper_rows_628hc7a8.append(
        {
            "name": name_628hc7a8,
            "type": type(obj_628hc7a8).__name__,
            "signature": signature_628hc7a8,
            "fd002_name_hint": bool(
                "fd002" in name_lower_628hc7a8
            ),
        }
    )


FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8 = pd.DataFrame(
    helper_rows_628hc7a8
)

if len(
    FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
) > 0:

    FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8 = (
        FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
        .sort_values(
            ["fd002_name_hint", "name"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


print(
    "\n--- MC / DROPOUT HELPER CANDIDATES: SIGNATURES ONLY ---"
)

if len(
    FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
) == 0:

    print(
        "No MC-dropout helper candidate found."
    )

else:

    display(
        FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
    )


# --------------------------------------------------------------------------------------------------
# 4) Historical execution scalar references
# --------------------------------------------------------------------------------------------------

scalar_rows_628hc7a8 = []

scalar_tokens_628hc7a8 = (
    "model",
    "parameter",
    "param",
    "mc",
    "dropout",
    "passes",
    "pass",
    "seed",
    "batch",
)

for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    if not any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in scalar_tokens_628hc7a8
    ):
        continue

    if not (
        "fd002" in name_lower_628hc7a8
        or
        "628fb" in name_lower_628hc7a8
        or
        "628hc" in name_lower_628hc7a8
    ):
        continue

    if isinstance(
        obj_628hc7a8,
        (
            bool,
            int,
            float,
            str,
            np.bool_,
            np.integer,
            np.floating,
        ),
    ):

        value_text_628hc7a8 = repr(
            obj_628hc7a8
        )

        if len(value_text_628hc7a8) > 180:

            value_text_628hc7a8 = (
                value_text_628hc7a8[:177]
                +
                "..."
            )

        scalar_rows_628hc7a8.append(
            {
                "name": name_628hc7a8,
                "type": type(
                    obj_628hc7a8
                ).__name__,
                "value": value_text_628hc7a8,
            }
        )


FD002_C7A8_EXECUTION_SCALAR_REFERENCES_628HC7A8 = pd.DataFrame(
    scalar_rows_628hc7a8
)

print(
    "\n--- HISTORICAL EXECUTION SCALAR REFERENCES ---"
)

if len(
    FD002_C7A8_EXECUTION_SCALAR_REFERENCES_628HC7A8
) == 0:

    print(
        "No matching scalar execution references found."
    )

else:

    display(
        FD002_C7A8_EXECUTION_SCALAR_REFERENCES_628HC7A8
    )


# --------------------------------------------------------------------------------------------------
# 5) Stored SHA-256 / fingerprint references
# --------------------------------------------------------------------------------------------------

fingerprint_rows_628hc7a8 = []

hex_chars_628hc7a8 = set(
    "0123456789abcdefABCDEF"
)


def is_sha256_text_628hc7a8(value):

    if not isinstance(value, str):
        return False

    value_628hc7a8 = value.strip()

    return bool(
        len(value_628hc7a8) == 64
        and
        all(
            ch_628hc7a8 in hex_chars_628hc7a8
            for ch_628hc7a8
            in value_628hc7a8
        )
    )


for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    direct_name_hint_628hc7a8 = bool(
        (
            "fd002" in name_lower_628hc7a8
            or
            "model" in name_lower_628hc7a8
            or
            "mc" in name_lower_628hc7a8
        )
        and
        any(
            token_628hc7a8 in name_lower_628hc7a8
            for token_628hc7a8
            in (
                "sha",
                "hash",
                "fingerprint",
            )
        )
    )

    if (
        direct_name_hint_628hc7a8
        and
        is_sha256_text_628hc7a8(
            obj_628hc7a8
        )
    ):

        fingerprint_rows_628hc7a8.append(
            {
                "source": name_628hc7a8,
                "key": "<direct>",
                "sha256": obj_628hc7a8,
            }
        )

    if not isinstance(
        obj_628hc7a8,
        dict,
    ):
        continue

    container_name_hint_628hc7a8 = bool(
        "fd002" in name_lower_628hc7a8
        or
        "metadata" in name_lower_628hc7a8
        or
        "contract" in name_lower_628hc7a8
    )

    if not container_name_hint_628hc7a8:
        continue

    for (
        key_628hc7a8,
        value_628hc7a8,
    ) in obj_628hc7a8.items():

        key_lower_628hc7a8 = str(
            key_628hc7a8
        ).lower()

        relevant_key_628hc7a8 = bool(
            any(
                token_628hc7a8
                in
                key_lower_628hc7a8
                for token_628hc7a8
                in (
                    "sha",
                    "hash",
                    "fingerprint",
                )
            )
        )

        if (
            relevant_key_628hc7a8
            and
            is_sha256_text_628hc7a8(
                value_628hc7a8
            )
        ):

            fingerprint_rows_628hc7a8.append(
                {
                    "source": name_628hc7a8,
                    "key": str(
                        key_628hc7a8
                    ),
                    "sha256": value_628hc7a8,
                }
            )


FD002_C7A8_STORED_FINGERPRINT_REFERENCES_628HC7A8 = pd.DataFrame(
    fingerprint_rows_628hc7a8
)

print(
    "\n--- STORED SHA-256 / FINGERPRINT REFERENCES ---"
)

if len(
    FD002_C7A8_STORED_FINGERPRINT_REFERENCES_628HC7A8
) == 0:

    print(
        "No direct one-level stored SHA-256 references recovered."
    )

else:

    display(
        FD002_C7A8_STORED_FINGERPRINT_REFERENCES_628HC7A8
        .drop_duplicates()
        .reset_index(drop=True)
    )


# --------------------------------------------------------------------------------------------------
# 6) Historical pre-label endpoint ndarray candidates
#    LABEL / RUL / TARGET / ERROR objects explicitly excluded.
# --------------------------------------------------------------------------------------------------

output_rows_628hc7a8 = []

include_tokens_628hc7a8 = (
    "test",
    "fd002",
    "pred",
    "prediction",
    "mc",
    "std",
    "risk",
    "uncert",
    "accept",
    "abstain",
)

exclude_tokens_628hc7a8 = (
    "rul",
    "label",
    "target",
    "error",
    "mae",
    "score",
    "y_test",
    "ytest",
)

allowed_shapes_628hc7a8 = {
    (259,),
    (50, 259),
    (259, 50),
}


def array_sha256_628hc7a8(arr):

    contiguous_628hc7a8 = np.ascontiguousarray(
        arr
    )

    h_628hc7a8 = hashlib.sha256()

    h_628hc7a8.update(
        str(
            contiguous_628hc7a8.dtype
        ).encode("utf-8")
    )

    h_628hc7a8.update(
        np.asarray(
            contiguous_628hc7a8.shape,
            dtype=np.int64,
        ).tobytes()
    )

    h_628hc7a8.update(
        contiguous_628hc7a8.tobytes(
            order="C"
        )
    )

    return h_628hc7a8.hexdigest()


for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    if not isinstance(
        obj_628hc7a8,
        np.ndarray,
    ):
        continue

    shape_628hc7a8 = tuple(
        obj_628hc7a8.shape
    )

    if shape_628hc7a8 not in allowed_shapes_628hc7a8:
        continue

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    if not any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in include_tokens_628hc7a8
    ):
        continue

    if any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in exclude_tokens_628hc7a8
    ):
        continue

    try:

        finite_628hc7a8 = bool(
            np.isfinite(
                obj_628hc7a8
            ).all()
        )

    except Exception:

        finite_628hc7a8 = None

    if (
        np.issubdtype(
            obj_628hc7a8.dtype,
            np.number,
        )
        or
        np.issubdtype(
            obj_628hc7a8.dtype,
            np.bool_,
        )
    ):

        numeric_628hc7a8 = obj_628hc7a8.astype(
            np.float64,
            copy=False,
        )

        minimum_628hc7a8 = float(
            np.min(
                numeric_628hc7a8
            )
        )

        median_628hc7a8 = float(
            np.median(
                numeric_628hc7a8
            )
        )

        mean_628hc7a8 = float(
            np.mean(
                numeric_628hc7a8
            )
        )

        maximum_628hc7a8 = float(
            np.max(
                numeric_628hc7a8
            )
        )

    else:

        minimum_628hc7a8 = np.nan
        median_628hc7a8 = np.nan
        mean_628hc7a8 = np.nan
        maximum_628hc7a8 = np.nan

    output_rows_628hc7a8.append(
        {
            "name": name_628hc7a8,
            "shape": shape_628hc7a8,
            "dtype": str(
                obj_628hc7a8.dtype
            ),
            "finite": finite_628hc7a8,
            "min": minimum_628hc7a8,
            "median": median_628hc7a8,
            "mean": mean_628hc7a8,
            "max": maximum_628hc7a8,
            "content_sha256":
                array_sha256_628hc7a8(
                    obj_628hc7a8
                ),
        }
    )


FD002_C7A8_PRELABEL_ARRAY_CANDIDATES_628HC7A8 = pd.DataFrame(
    output_rows_628hc7a8
)

print(
    "\n--- HISTORICAL PRE-LABEL ENDPOINT ARRAY CANDIDATES ---"
)

if len(
    FD002_C7A8_PRELABEL_ARRAY_CANDIDATES_628HC7A8
) == 0:

    print(
        "No shape-(259,) / MC endpoint output candidates found."
    )

else:

    display(
        FD002_C7A8_PRELABEL_ARRAY_CANDIDATES_628HC7A8
        .sort_values(
            ["shape", "name"],
            key=lambda s: s.astype(str),
        )
        .reset_index(drop=True)
    )


# --------------------------------------------------------------------------------------------------
# 7) Historical 259-row pre-label DataFrame candidates
# --------------------------------------------------------------------------------------------------

dataframe_rows_628hc7a8 = []

for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    if not isinstance(
        obj_628hc7a8,
        pd.DataFrame,
    ):
        continue

    if obj_628hc7a8.shape[0] != 259:
        continue

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    if any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in exclude_tokens_628hc7a8
    ):
        continue

    if not any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in (
            "fd002",
            "test",
            "decision",
            "risk",
            "accept",
            "pred",
        )
    ):
        continue

    dataframe_rows_628hc7a8.append(
        {
            "name": name_628hc7a8,
            "shape": tuple(
                obj_628hc7a8.shape
            ),
            "columns": list(
                obj_628hc7a8.columns
            ),
        }
    )


FD002_C7A8_PRELABEL_DATAFRAME_CANDIDATES_628HC7A8 = pd.DataFrame(
    dataframe_rows_628hc7a8
)

print(
    "\n--- HISTORICAL 259-ROW PRE-LABEL DATAFRAME CANDIDATES ---"
)

if len(
    FD002_C7A8_PRELABEL_DATAFRAME_CANDIDATES_628HC7A8
) == 0:

    print(
        "No 259-row pre-label DataFrame candidates found."
    )

else:

    display(
        FD002_C7A8_PRELABEL_DATAFRAME_CANDIDATES_628HC7A8
    )


# --------------------------------------------------------------------------------------------------
# 8) Label / RUL namespace guard
#
# IMPORTANT:
# We inspect NAMES / TYPES / SHAPE METADATA ONLY.
# We do NOT inspect any possible label values.
# --------------------------------------------------------------------------------------------------

label_guard_rows_628hc7a8 = []

for name_628hc7a8, obj_628hc7a8 in global_items_628hc7a8:

    name_lower_628hc7a8 = str(
        name_628hc7a8
    ).lower()

    if not any(
        token_628hc7a8 in name_lower_628hc7a8
        for token_628hc7a8
        in (
            "rul_fd002",
            "fd002_rul",
            "test_label",
            "test_target",
            "official_rul",
        )
    ):
        continue

    shape_text_628hc7a8 = None

    if hasattr(
        obj_628hc7a8,
        "shape",
    ):

        try:

            shape_text_628hc7a8 = tuple(
                obj_628hc7a8.shape
            )

        except Exception:

            shape_text_628hc7a8 = None

    label_guard_rows_628hc7a8.append(
        {
            "name": name_628hc7a8,
            "type": type(
                obj_628hc7a8
            ).__name__,
            "shape_metadata_only":
                shape_text_628hc7a8,
        }
    )


FD002_C7A8_LABEL_NAMESPACE_GUARD_628HC7A8 = pd.DataFrame(
    label_guard_rows_628hc7a8
)

print(
    "\n--- LABEL / RUL NAMESPACE GUARD: NAMES + TYPES ONLY ---"
)

if len(
    FD002_C7A8_LABEL_NAMESPACE_GUARD_628HC7A8
) == 0:

    print(
        "No matching label/RUL-like in-memory object names detected."
    )

else:

    display(
        FD002_C7A8_LABEL_NAMESPACE_GUARD_628HC7A8
    )


# --------------------------------------------------------------------------------------------------
# 9) Audit
# --------------------------------------------------------------------------------------------------

model_candidate_found_628hc7a8 = bool(
    len(
        FD002_C7A8_MODEL_CANDIDATES_628HC7A8
    ) >= 1
)

mc_helper_found_628hc7a8 = bool(
    len(
        FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
    ) >= 1
)

historical_prelabel_output_found_628hc7a8 = bool(
    len(
        FD002_C7A8_PRELABEL_ARRAY_CANDIDATES_628HC7A8
    ) >= 1
    or
    len(
        FD002_C7A8_PRELABEL_DATAFRAME_CANDIDATES_628HC7A8
    ) >= 1
)

audit_628hc7a8 = pd.DataFrame(
    {
        "check": [

            "previous_C7_A7_passed",

            "locked_test_tensor_shape_259x30x17",

            "locked_test_tensor_exact_historical",

            "torch_model_candidate_found",

            "mc_dropout_helper_candidate_found",

            "historical_prelabel_output_candidate_found",

            "official_rul_not_opened_in_this_step",

            "test_not_scored",

            "no_model_forward_pass",

            "no_mc_dropout_execution",

            "no_model_retraining",

            "no_dev_retuning",

            "no_threshold_or_rule_modification",

            "no_files_written",
        ],

        "passed": [

            True,

            bool(
                locked_x_628hc7a8.shape
                ==
                (259, 30, 17)
            ),

            bool(
                np.array_equal(
                    locked_x_628hc7a8,
                    np.asarray(
                        X_TEST_FD002_628D
                    ),
                )
            ),

            model_candidate_found_628hc7a8,

            mc_helper_found_628hc7a8,

            historical_prelabel_output_found_628hc7a8,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)

print(
    "\n--- C7-A8 AUDIT ---"
)

display(
    audit_628hc7a8
)

all_checks_passed_628hc7a8 = bool(
    audit_628hc7a8[
        "passed"
    ].all()
)

if not all_checks_passed_628hc7a8:

    raise RuntimeError(
        "C7-A8 discovery contract did not pass. "
        "Do not execute FD002 model/MC dropout yet."
    )


# --------------------------------------------------------------------------------------------------
# 10) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 126
)

print(
    "STEP 6.28H-C7-A8 FINAL SUMMARY"
)

print(
    "=" * 126
)

print(
    "Locked TEST endpoint tensor exact        :",
    True,
)

print(
    "Torch model candidates found             :",
    len(
        FD002_C7A8_MODEL_CANDIDATES_628HC7A8
    ),
)

print(
    "MC/dropout helper candidates found       :",
    len(
        FD002_C7A8_MC_HELPER_CANDIDATES_628HC7A8
    ),
)

print(
    "Historical endpoint array candidates     :",
    len(
        FD002_C7A8_PRELABEL_ARRAY_CANDIDATES_628HC7A8
    ),
)

print(
    "Historical 259-row DataFrame candidates  :",
    len(
        FD002_C7A8_PRELABEL_DATAFRAME_CANDIDATES_628HC7A8
    ),
)

print(
    "Official RUL opened in this step         :",
    False,
)

print(
    "TEST scored                              :",
    False,
)

print(
    "Model forward pass executed              :",
    False,
)

print(
    "MC dropout executed                      :",
    False,
)

print(
    "DEV retuning                             :",
    False,
)

print(
    "Threshold/rule modified                  :",
    False,
)

print(
    "Files written                            :",
    False,
)

print(
    "All C7-A8 checks passed                  :",
    all_checks_passed_628hc7a8,
)

print(
    "\nPASS — the exact FD002 model/helper/output candidates are now exposed "
    "without executing the model or touching official RUL contents."
)

print(
    "Next: resolve one authoritative model + one authoritative MC helper, "
    "recover their exact historical execution parameters, and verify any "
    "historical pre-label endpoint outputs before the one-way frozen TEST execution."
)

print(
    "\n=== END STEP 6.28H-C7-A8 ==="
)

In [ ]:
# ==================================================================================================
# STEP 6.28H-C7-A9 — FD002 PRE-LABEL ARTIFACT + HISTORICAL EXECUTION-EVIDENCE LOCK
#
# PURPOSE
# -------
# Lock the already-existing FD002 pre-label prediction/risk/decision artifacts BEFORE looking at
# any historical post-label scoring object. A8 exposed a complete historical C5 pre-scoring chain,
# so we VERIFY that chain rather than re-running stochastic inference.
#
# SAFETY
# ------
# - Does NOT open RUL_FD002.txt.
# - Does NOT inspect FD002_TEST_SCORING_TABLE_628HC5S1 values.
# - Does NOT score TEST.
# - Does NOT run the model.
# - Does NOT run MC dropout.
# - Does NOT retrain/retune anything.
# - Does NOT modify the frozen threshold/rule.
# - Writes no files.
# ==================================================================================================

import inspect
import hashlib
import numpy as np
import pandas as pd
import torch

print("=" * 128)
print("STEP 6.28H-C7-A9 — FD002 PRE-LABEL ARTIFACT + HISTORICAL EXECUTION-EVIDENCE LOCK")
print("=" * 128)

# --------------------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a8", False)):
    raise RuntimeError("C7-A8 must pass before C7-A9.")

required_628hc7a9 = [
    "MODEL_FD002_628EB",
    "TEST_MC_MATRIX_FD002_628FAR2",
    "FD002_CANONICAL_TEST_PREDICTION_628HC5F2A",
    "FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2",
    "FD002_C5_RELOADED_CANONICAL_TEST_RISK_628HC5F1B",
    "FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B",
    "FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
    "FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B",
]

missing_628hc7a9 = [
    name
    for name in required_628hc7a9
    if name not in globals()
]

if missing_628hc7a9:
    raise RuntimeError(
        "Missing required historical FD002 object(s): "
        + ", ".join(missing_628hc7a9)
    )

G_628hc7a9 = dict(globals())


# --------------------------------------------------------------------------------------------------
# 2) Authoritative historical model metadata — NO forward pass
# --------------------------------------------------------------------------------------------------

model_628hc7a9 = MODEL_FD002_628EB

if not isinstance(model_628hc7a9, torch.nn.Module):
    raise RuntimeError(
        "MODEL_FD002_628EB is not a torch.nn.Module."
    )

model_parameter_count_628hc7a9 = int(
    sum(
        p.numel()
        for p in model_628hc7a9.parameters()
    )
)

model_eval_628hc7a9 = bool(
    not model_628hc7a9.training
)

gru_input_sizes_628hc7a9 = [
    int(m.input_size)
    for m in model_628hc7a9.modules()
    if isinstance(m, torch.nn.GRU)
]


def model_state_sha256_628hc7a9(model):

    h = hashlib.sha256()

    for key, tensor in sorted(
        model.state_dict().items(),
        key=lambda item: item[0],
    ):

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h.update(
            key.encode("utf-8")
        )

        h.update(
            str(arr.dtype).encode("utf-8")
        )

        h.update(
            np.asarray(
                arr.shape,
                dtype=np.int64,
            ).tobytes()
        )

        h.update(
            arr.tobytes(
                order="C"
            )
        )

    return h.hexdigest()


model_state_sha_628hc7a9 = (
    model_state_sha256_628hc7a9(
        model_628hc7a9
    )
)

print(
    "\n--- AUTHORITATIVE HISTORICAL MODEL EVIDENCE ---"
)

print(
    "Model object       : MODEL_FD002_628EB"
)

print(
    "Model class        :",
    model_628hc7a9.__class__.__name__,
)

print(
    "Parameter count    :",
    model_parameter_count_628hc7a9,
)

print(
    "Eval mode          :",
    model_eval_628hc7a9,
)

print(
    "GRU input size(s)  :",
    gru_input_sizes_628hc7a9,
)

print(
    "State SHA-256      :",
    model_state_sha_628hc7a9,
)


# --------------------------------------------------------------------------------------------------
# 3) Historical MC evidence — NO MC execution
# --------------------------------------------------------------------------------------------------

mc_matrix_628hc7a9 = np.asarray(
    TEST_MC_MATRIX_FD002_628FAR2
)

if mc_matrix_628hc7a9.shape != (
    50,
    259,
):
    raise RuntimeError(
        "TEST_MC_MATRIX_FD002_628FAR2 "
        "does not have shape (50, 259)."
    )

if not np.isfinite(
    mc_matrix_628hc7a9
).all():
    raise RuntimeError(
        "Historical FD002 MC matrix contains "
        "non-finite values."
    )

mc_std_from_matrix_628hc7a9 = np.std(
    mc_matrix_628hc7a9.astype(
        np.float64,
        copy=False,
    ),
    axis=0,
    ddof=0,
)

mc_std_rows_628hc7a9 = []

for name, obj in G_628hc7a9.items():

    if not (
        isinstance(
            obj,
            np.ndarray,
        )
        and
        tuple(
            obj.shape
        )
        ==
        (259,)
    ):
        continue

    lower_name = str(
        name
    ).lower()

    if not (
        "mc" in lower_name
        and
        "std" in lower_name
    ):
        continue

    arr = np.asarray(
        obj,
        dtype=np.float64,
    )

    max_abs_difference = float(
        np.max(
            np.abs(
                arr
                -
                mc_std_from_matrix_628hc7a9
            )
        )
    )

    mc_std_rows_628hc7a9.append(
        {
            "name":
                name,

            "dtype":
                str(
                    obj.dtype
                ),

            "max_abs_difference_vs_TEST_MC_MATRIX_std":
                max_abs_difference,

            "matches_atol_1e_5":
                bool(
                    np.allclose(
                        arr,
                        mc_std_from_matrix_628hc7a9,
                        rtol=0.0,
                        atol=1e-5,
                    )
                ),
        }
    )

FD002_C7A9_MC_STD_LINEAGE_628HC7A9 = pd.DataFrame(
    mc_std_rows_628hc7a9
)

if len(
    FD002_C7A9_MC_STD_LINEAGE_628HC7A9
) > 0:

    FD002_C7A9_MC_STD_LINEAGE_628HC7A9 = (
        FD002_C7A9_MC_STD_LINEAGE_628HC7A9
        .sort_values(
            [
                "matches_atol_1e_5",
                "max_abs_difference_vs_TEST_MC_MATRIX_std",
                "name",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

helper_signature_628hc7a9 = None

if (
    "mc_dropout_predict_628hc0"
    in
    G_628hc7a9
):

    helper_signature_628hc7a9 = str(
        inspect.signature(
            G_628hc7a9[
                "mc_dropout_predict_628hc0"
            ]
        )
    )

print(
    "\n--- HISTORICAL MC EVIDENCE ---"
)

print(
    "Historical MC matrix shape :",
    mc_matrix_628hc7a9.shape,
)

print(
    "Historical MC pass count   :",
    mc_matrix_628hc7a9.shape[0],
)

print(
    "MC STD convention          : ddof=0"
)

print(
    "Later frozen helper sig.   :",
    helper_signature_628hc7a9,
)

if len(
    FD002_C7A9_MC_STD_LINEAGE_628HC7A9
) > 0:

    display(
        FD002_C7A9_MC_STD_LINEAGE_628HC7A9
    )


# --------------------------------------------------------------------------------------------------
# 4) Canonical historical pre-label prediction
# --------------------------------------------------------------------------------------------------

prediction_628hc7a9 = np.asarray(
    FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,
    dtype=np.float64,
)

if prediction_628hc7a9.shape != (
    259,
):
    raise RuntimeError(
        "Canonical FD002 TEST prediction "
        "is not shape (259,)."
    )

if not np.isfinite(
    prediction_628hc7a9
).all():
    raise RuntimeError(
        "Canonical FD002 TEST prediction "
        "contains non-finite values."
    )

prediction_rows_628hc7a9 = []

for name, obj in G_628hc7a9.items():

    if not (
        isinstance(
            obj,
            np.ndarray,
        )
        and
        tuple(
            obj.shape
        )
        ==
        (259,)
    ):
        continue

    lower_name = str(
        name
    ).lower()

    if not (
        "pred" in lower_name
        or
        "prediction" in lower_name
    ):
        continue

    if any(
        token in lower_name
        for token
        in (
            "error",
            "label",
            "target",
            "rul",
            "score",
        )
    ):
        continue

    arr = np.asarray(
        obj,
        dtype=np.float64,
    )

    prediction_rows_628hc7a9.append(
        {
            "name":
                name,

            "exact_to_canonical_prediction":
                bool(
                    np.array_equal(
                        arr,
                        prediction_628hc7a9,
                    )
                ),

            "max_abs_difference":
                float(
                    np.max(
                        np.abs(
                            arr
                            -
                            prediction_628hc7a9
                        )
                    )
                ),
        }
    )

FD002_C7A9_PREDICTION_LINEAGE_628HC7A9 = pd.DataFrame(
    prediction_rows_628hc7a9
)

if len(
    FD002_C7A9_PREDICTION_LINEAGE_628HC7A9
) > 0:

    FD002_C7A9_PREDICTION_LINEAGE_628HC7A9 = (
        FD002_C7A9_PREDICTION_LINEAGE_628HC7A9
        .sort_values(
            [
                "exact_to_canonical_prediction",
                "max_abs_difference",
                "name",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

print(
    "\n--- CANONICAL PRE-LABEL PREDICTION IDENTITY ---"
)

display(
    FD002_C7A9_PREDICTION_LINEAGE_628HC7A9
    .head(20)
)


# --------------------------------------------------------------------------------------------------
# 5) Frozen reliability risk + accept/abstain decision identity
# --------------------------------------------------------------------------------------------------

risk_628hc7a9 = np.asarray(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2,
    dtype=np.float64,
)

risk_reload_628hc7a9 = np.asarray(
    FD002_C5_RELOADED_CANONICAL_TEST_RISK_628HC5F1B,
    dtype=np.float64,
)

accept_reload_628hc7a9 = np.asarray(
    FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B,
    dtype=bool,
)

abstain_reload_628hc7a9 = np.asarray(
    FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B,
    dtype=bool,
)

threshold_628hc7a9 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

risk_contract_ok_628hc7a9 = bool(
    risk_628hc7a9.shape
    ==
    (259,)
    and
    np.isfinite(
        risk_628hc7a9
    ).all()
    and
    (
        (
            risk_628hc7a9 >= 0.0
        )
        &
        (
            risk_628hc7a9 <= 1.0
        )
    ).all()
)

expected_accept_628hc7a9 = (
    risk_628hc7a9
    <=
    threshold_628hc7a9
)

expected_abstain_628hc7a9 = (
    ~expected_accept_628hc7a9
)

risk_reload_exact_628hc7a9 = bool(
    np.array_equal(
        risk_628hc7a9,
        risk_reload_628hc7a9,
    )
)

accept_exact_628hc7a9 = bool(
    np.array_equal(
        expected_accept_628hc7a9,
        accept_reload_628hc7a9,
    )
)

abstain_exact_628hc7a9 = bool(
    np.array_equal(
        expected_abstain_628hc7a9,
        abstain_reload_628hc7a9,
    )
)

masks_complementary_628hc7a9 = bool(
    np.array_equal(
        abstain_reload_628hc7a9,
        ~accept_reload_628hc7a9,
    )
)

accepted_count_628hc7a9 = int(
    accept_reload_628hc7a9.sum()
)

abstained_count_628hc7a9 = int(
    abstain_reload_628hc7a9.sum()
)

print(
    "\n--- FROZEN RISK / DECISION IDENTITY ---"
)

print(
    "Frozen threshold             :",
    threshold_628hc7a9,
)

print(
    "Risk exact persisted reload  :",
    risk_reload_exact_628hc7a9,
)

print(
    "Accept exact threshold rule  :",
    accept_exact_628hc7a9,
)

print(
    "Abstain exact complement     :",
    abstain_exact_628hc7a9,
)

print(
    "Masks complementary          :",
    masks_complementary_628hc7a9,
)

print(
    "Accepted / abstained         :",
    accepted_count_628hc7a9,
    "/",
    abstained_count_628hc7a9,
)

print(
    "Realized pre-label coverage  :",
    accepted_count_628hc7a9
    /
    259.0,
)


# --------------------------------------------------------------------------------------------------
# 6) Historical pre-scoring contract structure
# --------------------------------------------------------------------------------------------------

prescoring_contract_628hc7a9 = (
    FD002_TEST_PRESCORING_CONTRACT_628HC5A5
)

if not isinstance(
    prescoring_contract_628hc7a9,
    dict,
):
    raise RuntimeError(
        "FD002_TEST_PRESCORING_CONTRACT_628HC5A5 "
        "is not a dict."
    )

required_hash_keys_628hc7a9 = [
    "risk_sha256",
    "prediction_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
    "prescoring_contract_sha256",
]


def valid_sha256_text_628hc7a9(
    value
):

    if not isinstance(
        value,
        str,
    ):
        return False

    value = value.strip()

    if len(
        value
    ) != 64:
        return False

    try:
        int(
            value,
            16,
        )
    except ValueError:
        return False

    return True


prescoring_hash_contract_valid_628hc7a9 = bool(
    all(
        key
        in
        prescoring_contract_628hc7a9
        and
        valid_sha256_text_628hc7a9(
            prescoring_contract_628hc7a9[
                key
            ]
        )
        for key
        in
        required_hash_keys_628hc7a9
    )
)

print(
    "\n--- HISTORICAL PRE-SCORING CONTRACT ---"
)

print(
    "Required SHA fields valid :",
    prescoring_hash_contract_valid_628hc7a9,
)


# --------------------------------------------------------------------------------------------------
# 7) Fresh in-memory PRE-LABEL fingerprints
#
# These are solely for immutability verification after any later evaluation-only label access.
# --------------------------------------------------------------------------------------------------

def fingerprint_array_628hc7a9(
    arr
):

    arr = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    h.update(
        str(
            arr.dtype
        ).encode(
            "utf-8"
        )
    )

    h.update(
        np.asarray(
            arr.shape,
            dtype=np.int64,
        ).tobytes()
    )

    h.update(
        arr.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


FD002_C7A9_FROZEN_PREDICTION_628HC7A9 = (
    prediction_628hc7a9.copy()
)

FD002_C7A9_FROZEN_RISK_628HC7A9 = (
    risk_628hc7a9.copy()
)

FD002_C7A9_FROZEN_ACCEPT_MASK_628HC7A9 = (
    accept_reload_628hc7a9.copy()
)

FD002_C7A9_FROZEN_ABSTAIN_MASK_628HC7A9 = (
    abstain_reload_628hc7a9.copy()
)

FD002_C7A9_PRELABEL_PREDICTION_SHA256_628HC7A9 = (
    fingerprint_array_628hc7a9(
        FD002_C7A9_FROZEN_PREDICTION_628HC7A9
    )
)

FD002_C7A9_PRELABEL_RISK_SHA256_628HC7A9 = (
    fingerprint_array_628hc7a9(
        FD002_C7A9_FROZEN_RISK_628HC7A9
    )
)

FD002_C7A9_PRELABEL_ACCEPT_SHA256_628HC7A9 = (
    fingerprint_array_628hc7a9(
        FD002_C7A9_FROZEN_ACCEPT_MASK_628HC7A9
    )
)

FD002_C7A9_PRELABEL_ABSTAIN_SHA256_628HC7A9 = (
    fingerprint_array_628hc7a9(
        FD002_C7A9_FROZEN_ABSTAIN_MASK_628HC7A9
    )
)

FD002_C7A9_PRELABEL_THRESHOLD_628HC7A9 = float(
    threshold_628hc7a9
)

FD002_C7A9_PRELABEL_MODEL_STATE_SHA256_628HC7A9 = (
    model_state_sha_628hc7a9
)

print(
    "\n--- FRESH PRE-LABEL IMMUTABILITY FINGERPRINTS ---"
)

print(
    "Prediction :",
    FD002_C7A9_PRELABEL_PREDICTION_SHA256_628HC7A9,
)

print(
    "Risk       :",
    FD002_C7A9_PRELABEL_RISK_SHA256_628HC7A9,
)

print(
    "Accept     :",
    FD002_C7A9_PRELABEL_ACCEPT_SHA256_628HC7A9,
)

print(
    "Abstain    :",
    FD002_C7A9_PRELABEL_ABSTAIN_SHA256_628HC7A9,
)

print(
    "Model      :",
    FD002_C7A9_PRELABEL_MODEL_STATE_SHA256_628HC7A9,
)


# --------------------------------------------------------------------------------------------------
# 8) Audit
# --------------------------------------------------------------------------------------------------

mc_std_lineage_ok_628hc7a9 = bool(
    len(
        FD002_C7A9_MC_STD_LINEAGE_628HC7A9
    ) > 0
    and
    FD002_C7A9_MC_STD_LINEAGE_628HC7A9[
        "matches_atol_1e_5"
    ].any()
)

prediction_lineage_ok_628hc7a9 = bool(
    len(
        FD002_C7A9_PREDICTION_LINEAGE_628HC7A9
    ) > 0
    and
    FD002_C7A9_PREDICTION_LINEAGE_628HC7A9[
        "exact_to_canonical_prediction"
    ].sum()
    >=
    2
)

audit_628hc7a9 = pd.DataFrame(
    {
        "check": [
            "previous_C7_A8_passed",
            "authoritative_model_parameter_count_18049",
            "authoritative_model_eval_mode",
            "authoritative_model_GRU_input_size_17",
            "historical_mc_matrix_shape_50x259",
            "historical_mc_std_lineage_recovered",
            "canonical_prediction_shape_259_finite",
            "canonical_prediction_alias_identity_recovered",
            "canonical_risk_shape_259_finite_in_0_1",
            "canonical_risk_exact_persisted_reload",
            "accept_mask_exact_frozen_threshold_rule",
            "abstain_mask_exact_complement",
            "accepted_plus_abstained_259",
            "historical_prescoring_hash_contract_valid",
            "fresh_prelabel_fingerprints_locked",
            "official_rul_not_opened_in_this_step",
            "historical_scoring_table_values_not_inspected",
            "test_not_scored_in_this_step",
            "no_model_forward_pass",
            "no_mc_dropout_execution",
            "no_model_retraining",
            "no_dev_retuning",
            "no_threshold_or_rule_modification",
            "no_files_written",
        ],

        "passed": [
            True,

            bool(
                model_parameter_count_628hc7a9
                ==
                18049
            ),

            model_eval_628hc7a9,

            bool(
                17
                in
                gru_input_sizes_628hc7a9
            ),

            bool(
                mc_matrix_628hc7a9.shape
                ==
                (50, 259)
            ),

            mc_std_lineage_ok_628hc7a9,

            bool(
                prediction_628hc7a9.shape
                ==
                (259,)
                and
                np.isfinite(
                    prediction_628hc7a9
                ).all()
            ),

            prediction_lineage_ok_628hc7a9,

            risk_contract_ok_628hc7a9,

            risk_reload_exact_628hc7a9,

            accept_exact_628hc7a9,

            bool(
                abstain_exact_628hc7a9
                and
                masks_complementary_628hc7a9
            ),

            bool(
                accepted_count_628hc7a9
                +
                abstained_count_628hc7a9
                ==
                259
            ),

            prescoring_hash_contract_valid_628hc7a9,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,

            True,
        ],
    }
)

print(
    "\n--- C7-A9 AUDIT ---"
)

display(
    audit_628hc7a9
)

all_checks_passed_628hc7a9 = bool(
    audit_628hc7a9[
        "passed"
    ].all()
)

if not all_checks_passed_628hc7a9:
    raise RuntimeError(
        "C7-A9 did not pass. "
        "Do NOT inspect historical FD002 scoring values yet."
    )


# --------------------------------------------------------------------------------------------------
# 9) Final summary
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 128
)

print(
    "STEP 6.28H-C7-A9 FINAL SUMMARY"
)

print(
    "=" * 128
)

print(
    "Authoritative historical model     : MODEL_FD002_628EB"
)

print(
    "Model parameter count              :",
    model_parameter_count_628hc7a9,
)

print(
    "Model eval mode                    :",
    model_eval_628hc7a9,
)

print(
    "Historical MC matrix               :",
    mc_matrix_628hc7a9.shape,
)

print(
    "Historical MC passes               :",
    mc_matrix_628hc7a9.shape[0],
)

print(
    "Canonical prediction locked        :",
    prediction_628hc7a9.shape,
)

print(
    "Canonical reliability risk locked  :",
    risk_628hc7a9.shape,
)

print(
    "Frozen risk threshold              :",
    threshold_628hc7a9,
)

print(
    "Accepted / abstained               :",
    accepted_count_628hc7a9,
    "/",
    abstained_count_628hc7a9,
)

print(
    "Pre-label realized coverage        :",
    accepted_count_628hc7a9
    /
    259.0,
)

print(
    "Persisted risk reload exact        :",
    risk_reload_exact_628hc7a9,
)

print(
    "Frozen decision rule exact         :",
    bool(
        accept_exact_628hc7a9
        and
        abstain_exact_628hc7a9
        and
        masks_complementary_628hc7a9
    ),
)

print(
    "Historical prescoring contract OK  :",
    prescoring_hash_contract_valid_628hc7a9,
)

print(
    "Official RUL opened in this step   :",
    False,
)

print(
    "Scoring values inspected           :",
    False,
)

print(
    "Model/MC executed                  :",
    False,
)

print(
    "DEV retuning                       :",
    False,
)

print(
    "Threshold/rule modified            :",
    False,
)

print(
    "Files written                      :",
    False,
)

print(
    "All C7-A9 checks passed            :",
    all_checks_passed_628hc7a9,
)

print(
    "\nPASS — the authoritative FD002 pre-label prediction, reliability risk, "
    "and accept/abstain decisions are now frozen in memory with fresh fingerprints."
)

print(
    "A8 also exposed an existing historical post-label C5 scoring object. "
    "We have deliberately NOT inspected its values yet."
)

print(
    "Next: recover that historical FD002 final scoring evidence only after "
    "verifying these pre-label fingerprints remain unchanged. If the historical "
    "scoring chain is internally consistent, FD002 TEST does not need to be "
    "executed or read again."
)

print(
    "\n=== END STEP 6.28H-C7-A9 ==="
)

In [ ]:
# STEP 6.28H-C7-A9-R1
# Strict-whitelist FD002 model-state alias grouping (NO INFERENCE)

import pandas as pd
import torch

if not bool(globals().get("all_checks_passed_628hc7a9", False)):
    raise RuntimeError(
        "The previously executed C7-A9 cell must have completed before C7-A9-R1."
    )

if "model_state_sha256_628hc7a9" not in globals():
    raise RuntimeError(
        "Expected C7-A9 model-state fingerprint function is missing."
    )

C7A9R1_MODEL_WHITELIST = [
    "MODEL_FD002_628EB",
    "model",
    "model_628hc0",
    "model_628hc1a",
    "MODEL_FD001_628HC6A6A1",
    "FD001_MODEL_LOCKED_628HC6A7A2A0",
    "model_628hc6a7a2a1",
]

rows_628hc7a9r1 = []

for name_628hc7a9r1 in C7A9R1_MODEL_WHITELIST:
    obj_628hc7a9r1 = globals().get(name_628hc7a9r1, None)

    if obj_628hc7a9r1 is None:
        rows_628hc7a9r1.append(
            {
                "name": name_628hc7a9r1,
                "present": False,
                "is_torch_module": False,
                "class_name": None,
                "object_id": None,
                "parameter_count": None,
                "training_mode": None,
                "state_sha256": None,
            }
        )
        continue

    is_module_628hc7a9r1 = isinstance(obj_628hc7a9r1, torch.nn.Module)

    if not is_module_628hc7a9r1:
        rows_628hc7a9r1.append(
            {
                "name": name_628hc7a9r1,
                "present": True,
                "is_torch_module": False,
                "class_name": type(obj_628hc7a9r1).__name__,
                "object_id": id(obj_628hc7a9r1),
                "parameter_count": None,
                "training_mode": None,
                "state_sha256": None,
            }
        )
        continue

    rows_628hc7a9r1.append(
        {
            "name": name_628hc7a9r1,
            "present": True,
            "is_torch_module": True,
            "class_name": obj_628hc7a9r1.__class__.__name__,
            "object_id": id(obj_628hc7a9r1),
            "parameter_count": int(
                sum(p.numel() for p in obj_628hc7a9r1.parameters())
            ),
            "training_mode": bool(obj_628hc7a9r1.training),
            "state_sha256": model_state_sha256_628hc7a9(
                obj_628hc7a9r1
            ),
        }
    )

FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1 = pd.DataFrame(
    rows_628hc7a9r1
)

present_models_628hc7a9r1 = (
    FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1[
        FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1["present"]
        & FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1["is_torch_module"]
    ]
    .copy()
)

if len(present_models_628hc7a9r1) == 0:
    raise RuntimeError(
        "None of the strictly whitelisted historical model objects is available."
    )

FD002_C7A9R1_MODEL_STATE_GROUPS_628HC7A9R1 = (
    present_models_628hc7a9r1
    .groupby("state_sha256", dropna=False)
    .agg(
        count=("name", "size"),
        object_names=("name", list),
        object_ids=("object_id", list),
        classes=("class_name", list),
    )
    .reset_index()
    .sort_values(
        ["count", "state_sha256"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

fd002_row_628hc7a9r1 = (
    FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1[
        FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1["name"]
        == "MODEL_FD002_628EB"
    ]
)

if len(fd002_row_628hc7a9r1) != 1 or not bool(
    fd002_row_628hc7a9r1.iloc[0]["present"]
):
    raise RuntimeError(
        "MODEL_FD002_628EB is not available in the current notebook state."
    )

FD002_C7A9R1_FD002_NAMED_MODEL_SHA256_628HC7A9R1 = str(
    fd002_row_628hc7a9r1.iloc[0]["state_sha256"]
)

print("=" * 118)
print("STEP 6.28H-C7-A9-R1 — STRICT-WHITELIST MODEL-STATE ALIAS GROUPING")
print("=" * 118)
print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("No frozen model/policy/threshold state modified.")
print()

print("--- WHITELISTED MODEL OBJECTS ---")
display(FD002_C7A9R1_MODEL_ALIAS_AUDIT_628HC7A9R1)

print("\n--- EXACT STATE-DICT FINGERPRINT GROUPS ---")
display(FD002_C7A9R1_MODEL_STATE_GROUPS_628HC7A9R1)

print(
    "\nMODEL_FD002_628EB current state SHA-256 :",
    FD002_C7A9R1_FD002_NAMED_MODEL_SHA256_628HC7A9R1,
)

print(
    "\nC7-A9-R1 complete: model aliases grouped by exact current state only; "
    "authoritative FD002 model lineage is still NOT assigned by this cell."
)

In [ ]:
# STEP 6.28H-C7-A9-R2
# Strict-whitelist FD002 pre-label provenance-key + MC-helper signature inventory
# NO MODEL INFERENCE / NO ARRAY-VALUE INSPECTION

import inspect
import pandas as pd

if "FD002_C7A9R1_MODEL_STATE_GROUPS_628HC7A9R1" not in globals():
    raise RuntimeError(
        "C7-A9-R1 output is missing. Run/retain R1 before C7-A9-R2."
    )

# Exact pre-label metadata objects transferred in the handover.
C7A9R2_METADATA_WHITELIST = [
    "FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3",
    "FD002_C5_RELOADED_METADATA_628HC5F1B",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
    "metadata_628hc5f1a",
    "metadata_628hc5f2a",
    "metadata_reload_628hc5f2b",
]

# Exact MC-helper candidates recovered at C7-A8.
C7A9R2_HELPER_WHITELIST = [
    "mc_dropout_predict_628far2",
    "mc_dropout_predict_628fbr1",
    "mc_dropout_predict_628hc0",
]

metadata_rows_628hc7a9r2 = []

for name_628hc7a9r2 in C7A9R2_METADATA_WHITELIST:
    obj_628hc7a9r2 = globals().get(name_628hc7a9r2, None)

    if obj_628hc7a9r2 is None:
        metadata_rows_628hc7a9r2.append(
            {
                "name": name_628hc7a9r2,
                "present": False,
                "type": None,
                "top_level_key_count": None,
                "top_level_keys": None,
            }
        )
        continue

    if isinstance(obj_628hc7a9r2, dict):
        keys_628hc7a9r2 = sorted(
            [str(k) for k in obj_628hc7a9r2.keys()]
        )

        metadata_rows_628hc7a9r2.append(
            {
                "name": name_628hc7a9r2,
                "present": True,
                "type": "dict",
                "top_level_key_count": len(keys_628hc7a9r2),
                "top_level_keys": keys_628hc7a9r2,
            }
        )
    else:
        metadata_rows_628hc7a9r2.append(
            {
                "name": name_628hc7a9r2,
                "present": True,
                "type": type(obj_628hc7a9r2).__name__,
                "top_level_key_count": None,
                "top_level_keys": None,
            }
        )

FD002_C7A9R2_METADATA_KEY_INVENTORY_628HC7A9R2 = pd.DataFrame(
    metadata_rows_628hc7a9r2
)

helper_rows_628hc7a9r2 = []

for name_628hc7a9r2 in C7A9R2_HELPER_WHITELIST:
    obj_628hc7a9r2 = globals().get(name_628hc7a9r2, None)

    if obj_628hc7a9r2 is None:
        helper_rows_628hc7a9r2.append(
            {
                "name": name_628hc7a9r2,
                "present": False,
                "callable": False,
                "signature": None,
            }
        )
        continue

    is_callable_628hc7a9r2 = callable(obj_628hc7a9r2)

    signature_628hc7a9r2 = None
    if is_callable_628hc7a9r2:
        try:
            signature_628hc7a9r2 = str(
                inspect.signature(obj_628hc7a9r2)
            )
        except (TypeError, ValueError):
            signature_628hc7a9r2 = "<signature unavailable>"

    helper_rows_628hc7a9r2.append(
        {
            "name": name_628hc7a9r2,
            "present": True,
            "callable": is_callable_628hc7a9r2,
            "signature": signature_628hc7a9r2,
        }
    )

FD002_C7A9R2_HELPER_SIGNATURE_INVENTORY_628HC7A9R2 = pd.DataFrame(
    helper_rows_628hc7a9r2
)

print("=" * 118)
print(
    "STEP 6.28H-C7-A9-R2 — STRICT-WHITELIST PRE-LABEL "
    "PROVENANCE-KEY / MC-HELPER INVENTORY"
)
print("=" * 118)

print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No metadata values inspected; top-level key names only.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("No prediction/risk/mask array contents inspected.")
print("No frozen project state modified.")

print("\n--- WHITELISTED PRE-LABEL METADATA KEY INVENTORY ---")
for row_628hc7a9r2 in metadata_rows_628hc7a9r2:
    print("\nObject :", row_628hc7a9r2["name"])
    print("Present:", row_628hc7a9r2["present"])
    print("Type   :", row_628hc7a9r2["type"])

    if row_628hc7a9r2["top_level_keys"] is not None:
        print(
            "Key count:",
            row_628hc7a9r2["top_level_key_count"],
        )
        print("Keys:")
        for key_628hc7a9r2 in row_628hc7a9r2["top_level_keys"]:
            print("  -", key_628hc7a9r2)

print("\n--- EXACT MC-HELPER SIGNATURE INVENTORY ---")
print(
    FD002_C7A9R2_HELPER_SIGNATURE_INVENTORY_628HC7A9R2.to_string(
        index=False
    )
)

print(
    "\nC7-A9-R2 complete: provenance field locations inventoried only; "
    "no provenance value has yet been used to assign model/helper authority."
)

In [ ]:
# STEP 6.28H-C7-A9-R3
# Strict-whitelist historical pre-label metadata value + stored-hash consistency recovery
# NO MODEL INFERENCE / NO ARRAY-VALUE INSPECTION

import pandas as pd

if "FD002_C7A9R2_METADATA_KEY_INVENTORY_628HC7A9R2" not in globals():
    raise RuntimeError(
        "C7-A9-R2 output is missing. Run/retain R2 before C7-A9-R3."
    )

C7A9R3_METADATA_FIELDS = {
    "FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3": [
        "dataset",
        "reliability_architecture",
        "operating_policy",
        "canonical_test_risk_source",
        "canonical_test_risk_sha256",
        "decision_rule",
        "frozen_risk_threshold",
        "nominal_coverage",
        "population",
        "accepted_count",
        "abstained_count",
        "realized_coverage",
        "realized_abstention",
        "accept_mask_sha256",
        "abstain_mask_sha256",
        "official_test_labels_read",
        "test_scored",
    ],
    "FD002_C5_RELOADED_METADATA_628HC5F1B": [
        "step",
        "dataset",
        "reliability_architecture",
        "operating_policy",
        "canonical_test_risk_source",
        "canonical_test_risk_sha256",
        "decision_rule",
        "frozen_risk_threshold",
        "nominal_coverage",
        "population",
        "accepted_count",
        "abstained_count",
        "realized_coverage",
        "realized_abstention",
        "accept_mask_sha256",
        "abstain_mask_sha256",
        "model_executed",
        "model_retrained",
        "official_test_labels_read",
        "test_scored",
        "threshold_tuned_or_modified",
        "npz_filename",
        "npz_file_sha256",
        "decision_csv_filename",
        "decision_csv_sha256",
    ],
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5": [
        "n_test_endpoints",
        "canonical_prediction_source",
        "canonical_risk_source",
        "reliability_architecture",
        "operating_policy",
        "frozen_risk_threshold",
        "accepted_test_endpoints",
        "abstained_test_endpoints",
        "realized_test_coverage",
        "prediction_sha256",
        "risk_sha256",
        "accept_mask_sha256",
        "abstain_mask_sha256",
        "prescoring_contract_sha256",
    ],
    "metadata_628hc5f1a": [
        "step",
        "dataset",
        "reliability_architecture",
        "operating_policy",
        "canonical_test_risk_source",
        "canonical_test_risk_sha256",
        "decision_rule",
        "frozen_risk_threshold",
        "nominal_coverage",
        "population",
        "accepted_count",
        "abstained_count",
        "realized_coverage",
        "realized_abstention",
        "accept_mask_sha256",
        "abstain_mask_sha256",
        "model_executed",
        "model_retrained",
        "official_test_labels_read",
        "test_scored",
        "threshold_tuned_or_modified",
        "npz_filename",
        "npz_file_sha256",
        "decision_csv_filename",
        "decision_csv_sha256",
    ],
    "metadata_628hc5f2a": [
        "step",
        "dataset",
        "canonical_source_object",
        "deterministic_lineage_exact",
        "deterministic_lineage_objects",
        "mc_mean_used_as_point_prediction",
        "model_executed",
        "model_retrained",
        "n_test_endpoints",
        "official_test_labels_read",
        "test_scored",
        "threshold_tuned_or_modified",
        "prediction_dtype",
        "prediction_semantic_role",
        "prediction_shape",
        "prediction_sha256",
        "npz_filename",
        "npz_file_sha256",
        "csv_filename",
        "csv_file_sha256",
    ],
    "metadata_reload_628hc5f2b": [
        "step",
        "dataset",
        "canonical_source_object",
        "deterministic_lineage_exact",
        "deterministic_lineage_objects",
        "mc_mean_used_as_point_prediction",
        "model_executed",
        "model_retrained",
        "n_test_endpoints",
        "official_test_labels_read",
        "test_scored",
        "threshold_tuned_or_modified",
        "prediction_dtype",
        "prediction_semantic_role",
        "prediction_shape",
        "prediction_sha256",
        "npz_filename",
        "npz_file_sha256",
        "csv_filename",
        "csv_file_sha256",
    ],
}

def _safe_metadata_repr_628hc7a9r3(value):
    primitive_types = (str, bool, int, float, type(None))

    if isinstance(value, primitive_types):
        return repr(value)

    if isinstance(value, (list, tuple)):
        if all(isinstance(x, primitive_types) for x in value):
            return repr(value)
        return f"<{type(value).__name__}: non-primitive contents not expanded>"

    return f"<{type(value).__name__}: not expanded>"

rows_628hc7a9r3 = []

for object_name_628hc7a9r3, fields_628hc7a9r3 in C7A9R3_METADATA_FIELDS.items():
    obj_628hc7a9r3 = globals().get(object_name_628hc7a9r3, None)

    if not isinstance(obj_628hc7a9r3, dict):
        rows_628hc7a9r3.append(
            {
                "object": object_name_628hc7a9r3,
                "key": "<OBJECT>",
                "present": obj_628hc7a9r3 is not None,
                "value_type": (
                    None
                    if obj_628hc7a9r3 is None
                    else type(obj_628hc7a9r3).__name__
                ),
                "value": "<missing or not a dict>",
            }
        )
        continue

    for key_628hc7a9r3 in fields_628hc7a9r3:
        present_628hc7a9r3 = key_628hc7a9r3 in obj_628hc7a9r3

        value_628hc7a9r3 = (
            obj_628hc7a9r3[key_628hc7a9r3]
            if present_628hc7a9r3
            else None
        )

        rows_628hc7a9r3.append(
            {
                "object": object_name_628hc7a9r3,
                "key": key_628hc7a9r3,
                "present": present_628hc7a9r3,
                "value_type": (
                    type(value_628hc7a9r3).__name__
                    if present_628hc7a9r3
                    else None
                ),
                "value": (
                    _safe_metadata_repr_628hc7a9r3(value_628hc7a9r3)
                    if present_628hc7a9r3
                    else "<missing>"
                ),
            }
        )

FD002_C7A9R3_METADATA_VALUES_628HC7A9R3 = pd.DataFrame(
    rows_628hc7a9r3
)

C7A9R3_HASH_SOURCES = {
    "prediction_sha256": [
        ("FD002_TEST_PRESCORING_CONTRACT_628HC5A5", "prediction_sha256"),
        ("metadata_628hc5f2a", "prediction_sha256"),
        ("metadata_reload_628hc5f2b", "prediction_sha256"),
    ],
    "risk_sha256": [
        (
            "FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3",
            "canonical_test_risk_sha256",
        ),
        (
            "FD002_C5_RELOADED_METADATA_628HC5F1B",
            "canonical_test_risk_sha256",
        ),
        ("FD002_TEST_PRESCORING_CONTRACT_628HC5A5", "risk_sha256"),
        ("metadata_628hc5f1a", "canonical_test_risk_sha256"),
    ],
    "accept_mask_sha256": [
        ("FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3", "accept_mask_sha256"),
        ("FD002_C5_RELOADED_METADATA_628HC5F1B", "accept_mask_sha256"),
        ("FD002_TEST_PRESCORING_CONTRACT_628HC5A5", "accept_mask_sha256"),
        ("metadata_628hc5f1a", "accept_mask_sha256"),
    ],
    "abstain_mask_sha256": [
        ("FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3", "abstain_mask_sha256"),
        ("FD002_C5_RELOADED_METADATA_628HC5F1B", "abstain_mask_sha256"),
        ("FD002_TEST_PRESCORING_CONTRACT_628HC5A5", "abstain_mask_sha256"),
        ("metadata_628hc5f1a", "abstain_mask_sha256"),
    ],
}

hash_rows_628hc7a9r3 = []

for logical_name_628hc7a9r3, sources_628hc7a9r3 in C7A9R3_HASH_SOURCES.items():
    recovered_628hc7a9r3 = []

    for object_name_628hc7a9r3, key_628hc7a9r3 in sources_628hc7a9r3:
        obj_628hc7a9r3 = globals().get(object_name_628hc7a9r3, None)

        if (
            isinstance(obj_628hc7a9r3, dict)
            and key_628hc7a9r3 in obj_628hc7a9r3
        ):
            recovered_628hc7a9r3.append(
                (
                    object_name_628hc7a9r3,
                    key_628hc7a9r3,
                    obj_628hc7a9r3[key_628hc7a9r3],
                )
            )

    recovered_values_628hc7a9r3 = [
        value_628hc7a9r3
        for _, _, value_628hc7a9r3 in recovered_628hc7a9r3
    ]

    all_strings_628hc7a9r3 = bool(
        recovered_values_628hc7a9r3
        and all(
            isinstance(v, str)
            for v in recovered_values_628hc7a9r3
        )
    )

    all_equal_628hc7a9r3 = bool(
        all_strings_628hc7a9r3
        and len(set(recovered_values_628hc7a9r3)) == 1
    )

    hash_rows_628hc7a9r3.append(
        {
            "logical_artifact": logical_name_628hc7a9r3,
            "source_count": len(recovered_628hc7a9r3),
            "all_present_values_are_strings": all_strings_628hc7a9r3,
            "all_present_hashes_exactly_equal": all_equal_628hc7a9r3,
            "hash_value": (
                recovered_values_628hc7a9r3[0]
                if all_equal_628hc7a9r3
                else "<not uniquely consistent>"
            ),
            "sources": [
                f"{obj_name}.{key_name}"
                for obj_name, key_name, _ in recovered_628hc7a9r3
            ],
        }
    )

FD002_C7A9R3_STORED_HASH_CONSISTENCY_628HC7A9R3 = pd.DataFrame(
    hash_rows_628hc7a9r3
)

print("=" * 122)
print(
    "STEP 6.28H-C7-A9-R3 — STRICT-WHITELIST HISTORICAL PRE-LABEL "
    "METADATA VALUE + STORED-HASH CONSISTENCY RECOVERY"
)
print("=" * 122)

print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("No prediction/risk/mask array values inspected.")
print("Only explicitly whitelisted metadata fields were read.")
print("Non-primitive metadata contents are not expanded.")
print("No frozen project state modified.")

for object_name_628hc7a9r3 in C7A9R3_METADATA_FIELDS:
    print(f"\n--- {object_name_628hc7a9r3} ---")

    subset_628hc7a9r3 = FD002_C7A9R3_METADATA_VALUES_628HC7A9R3[
        FD002_C7A9R3_METADATA_VALUES_628HC7A9R3["object"]
        == object_name_628hc7a9r3
    ][["key", "present", "value_type", "value"]]

    print(subset_628hc7a9r3.to_string(index=False))

print("\n--- CROSS-METADATA STORED HASH CONSISTENCY ---")
print(
    FD002_C7A9R3_STORED_HASH_CONSISTENCY_628HC7A9R3.to_string(
        index=False
    )
)

print(
    "\nC7-A9-R3 complete: exact pre-label metadata values and stored-hash "
    "agreement recovered only; no array fingerprint recomputation or "
    "model/helper authority assignment performed."
)

In [ ]:
# STEP 6.28H-C7-A9-R4
# Exact historical pre-label artifact hash-convention recovery
# STRICT WHITELIST / NO INFERENCE / NO LABEL OR SCORING ACCESS

import hashlib
import io
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# Preconditions
# --------------------------------------------------------------------------------------

if "FD002_C7A9R3_STORED_HASH_CONSISTENCY_628HC7A9R3" not in globals():
    raise RuntimeError(
        "C7-A9-R3 output is missing. Retain/run R3 before C7-A9-R4."
    )

required_names_628hc7a9r4 = [
    "FD002_CANONICAL_TEST_PREDICTION_628HC5F2A",
    "FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2",
    "FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B",
    "FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
]

missing_628hc7a9r4 = [
    name
    for name in required_names_628hc7a9r4
    if name not in globals()
]

if missing_628hc7a9r4:
    raise RuntimeError(
        "Missing required pre-label object(s): "
        + ", ".join(missing_628hc7a9r4)
    )

contract_628hc7a9r4 = FD002_TEST_PRESCORING_CONTRACT_628HC5A5

if not isinstance(contract_628hc7a9r4, dict):
    raise RuntimeError(
        "FD002_TEST_PRESCORING_CONTRACT_628HC5A5 is not a dict."
    )

required_hash_keys_628hc7a9r4 = [
    "prediction_sha256",
    "risk_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
]

for key_628hc7a9r4 in required_hash_keys_628hc7a9r4:
    value_628hc7a9r4 = contract_628hc7a9r4.get(key_628hc7a9r4)

    if not (
        isinstance(value_628hc7a9r4, str)
        and len(value_628hc7a9r4.strip()) == 64
    ):
        raise RuntimeError(
            f"Historical hash field is invalid or missing: {key_628hc7a9r4}"
        )

# --------------------------------------------------------------------------------------
# Strictly whitelisted artifacts + historical hashes
# --------------------------------------------------------------------------------------

artifacts_628hc7a9r4 = {
    "prediction": {
        "array": FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,
        "historical_sha256": contract_628hc7a9r4["prediction_sha256"],
    },
    "risk": {
        "array": FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2,
        "historical_sha256": contract_628hc7a9r4["risk_sha256"],
    },
    "accept_mask": {
        "array": FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B,
        "historical_sha256": contract_628hc7a9r4["accept_mask_sha256"],
    },
    "abstain_mask": {
        "array": FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B,
        "historical_sha256": contract_628hc7a9r4["abstain_mask_sha256"],
    },
}


def _sha256_628hc7a9r4(payload):
    return hashlib.sha256(payload).hexdigest()


def _candidate_hashes_628hc7a9r4(value):
    arr = np.asarray(value)

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array encountered; refusing unsafe hash reconstruction."
        )

    arr_c = np.ascontiguousarray(arr)
    arr_f = np.asfortranarray(arr)

    raw_c = arr_c.tobytes(order="C")
    raw_f = arr_f.tobytes(order="F")

    dtype_text = str(arr_c.dtype).encode("utf-8")
    dtype_str = arr_c.dtype.str.encode("utf-8")

    shape_tuple_text = str(tuple(arr_c.shape)).encode("utf-8")
    shape_i64 = np.asarray(
        arr_c.shape,
        dtype=np.int64,
    ).tobytes()

    npy_buffer = io.BytesIO()
    np.save(
        npy_buffer,
        arr_c,
        allow_pickle=False,
    )
    npy_bytes = npy_buffer.getvalue()

    candidates = {
        # Most likely simple historical convention.
        "raw_C_bytes":
            _sha256_628hc7a9r4(raw_c),

        "raw_F_bytes":
            _sha256_628hc7a9r4(raw_f),

        "dtype_text_plus_raw":
            _sha256_628hc7a9r4(
                dtype_text
                + raw_c
            ),

        "shape_i64_plus_raw":
            _sha256_628hc7a9r4(
                shape_i64
                + raw_c
            ),

        "shape_tuple_text_plus_raw":
            _sha256_628hc7a9r4(
                shape_tuple_text
                + raw_c
            ),

        "dtype_text_plus_shape_i64_plus_raw":
            _sha256_628hc7a9r4(
                dtype_text
                + shape_i64
                + raw_c
            ),

        "dtype_text_plus_shape_tuple_plus_raw":
            _sha256_628hc7a9r4(
                dtype_text
                + shape_tuple_text
                + raw_c
            ),

        "dtype_str_plus_shape_tuple_plus_raw":
            _sha256_628hc7a9r4(
                dtype_str
                + shape_tuple_text
                + raw_c
            ),

        "dtype_pipe_shape_pipe_raw":
            _sha256_628hc7a9r4(
                dtype_text
                + b"|"
                + shape_tuple_text
                + b"|"
                + raw_c
            ),

        "numpy_npy_serialization":
            _sha256_628hc7a9r4(
                npy_bytes
            ),
    }

    return arr, candidates


rows_628hc7a9r4 = []
matches_by_artifact_628hc7a9r4 = {}

for artifact_name_628hc7a9r4, info_628hc7a9r4 in artifacts_628hc7a9r4.items():

    historical_sha_628hc7a9r4 = (
        info_628hc7a9r4["historical_sha256"]
        .strip()
        .lower()
    )

    arr_628hc7a9r4, candidates_628hc7a9r4 = (
        _candidate_hashes_628hc7a9r4(
            info_628hc7a9r4["array"]
        )
    )

    matching_conventions_628hc7a9r4 = []

    for convention_628hc7a9r4, sha_628hc7a9r4 in candidates_628hc7a9r4.items():

        exact_match_628hc7a9r4 = bool(
            sha_628hc7a9r4.lower()
            ==
            historical_sha_628hc7a9r4
        )

        if exact_match_628hc7a9r4:
            matching_conventions_628hc7a9r4.append(
                convention_628hc7a9r4
            )

        rows_628hc7a9r4.append(
            {
                "artifact": artifact_name_628hc7a9r4,
                "dtype": str(arr_628hc7a9r4.dtype),
                "shape": tuple(arr_628hc7a9r4.shape),
                "convention": convention_628hc7a9r4,
                "candidate_sha256": sha_628hc7a9r4,
                "historical_sha256": historical_sha_628hc7a9r4,
                "exact_match": exact_match_628hc7a9r4,
            }
        )

    matches_by_artifact_628hc7a9r4[
        artifact_name_628hc7a9r4
    ] = matching_conventions_628hc7a9r4


FD002_C7A9R4_HASH_CANDIDATE_AUDIT_628HC7A9R4 = pd.DataFrame(
    rows_628hc7a9r4
)

# --------------------------------------------------------------------------------------
# Determine whether one convention explains every historical artifact.
# --------------------------------------------------------------------------------------

match_sets_628hc7a9r4 = [
    set(matches)
    for matches in matches_by_artifact_628hc7a9r4.values()
]

if match_sets_628hc7a9r4:
    common_conventions_628hc7a9r4 = set.intersection(
        *match_sets_628hc7a9r4
    )
else:
    common_conventions_628hc7a9r4 = set()

FD002_C7A9R4_COMMON_HASH_CONVENTIONS_628HC7A9R4 = sorted(
    common_conventions_628hc7a9r4
)

FD002_C7A9R4_ALL_ARTIFACTS_HAVE_MATCH_628HC7A9R4 = bool(
    all(
        len(matches) > 0
        for matches in matches_by_artifact_628hc7a9r4.values()
    )
)

FD002_C7A9R4_ONE_COMMON_CONVENTION_EXISTS_628HC7A9R4 = bool(
    len(FD002_C7A9R4_COMMON_HASH_CONVENTIONS_628HC7A9R4) > 0
)

# --------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.28H-C7-A9-R4 — HISTORICAL PRE-LABEL "
    "ARTIFACT HASH-CONVENTION RECOVERY"
)
print("=" * 126)

print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("Only four explicitly whitelisted pre-label arrays were hashed.")
print("No frozen project state modified.")
print("No files written.")

for artifact_name_628hc7a9r4, info_628hc7a9r4 in artifacts_628hc7a9r4.items():

    artifact_rows_628hc7a9r4 = (
        FD002_C7A9R4_HASH_CANDIDATE_AUDIT_628HC7A9R4[
            FD002_C7A9R4_HASH_CANDIDATE_AUDIT_628HC7A9R4[
                "artifact"
            ]
            ==
            artifact_name_628hc7a9r4
        ]
    )

    first_row_628hc7a9r4 = artifact_rows_628hc7a9r4.iloc[0]

    print(f"\n--- {artifact_name_628hc7a9r4} ---")
    print("dtype          :", first_row_628hc7a9r4["dtype"])
    print("shape          :", first_row_628hc7a9r4["shape"])
    print(
        "historical SHA :",
        first_row_628hc7a9r4["historical_sha256"],
    )

    matching_rows_628hc7a9r4 = artifact_rows_628hc7a9r4[
        artifact_rows_628hc7a9r4["exact_match"]
    ]

    if len(matching_rows_628hc7a9r4) > 0:

        print("MATCHED convention(s):")

        for _, row_628hc7a9r4 in matching_rows_628hc7a9r4.iterrows():
            print(
                "  -",
                row_628hc7a9r4["convention"],
                "=>",
                row_628hc7a9r4["candidate_sha256"],
            )

    else:

        print("MATCHED convention(s): NONE")
        print("Candidate hashes tested:")

        for _, row_628hc7a9r4 in artifact_rows_628hc7a9r4.iterrows():
            print(
                "  -",
                row_628hc7a9r4["convention"],
                "=>",
                row_628hc7a9r4["candidate_sha256"],
            )


print("\n--- R4 SUMMARY ---")

print(
    "Every artifact matched at least one tested convention :",
    FD002_C7A9R4_ALL_ARTIFACTS_HAVE_MATCH_628HC7A9R4,
)

print(
    "One common convention explains all four artifacts     :",
    FD002_C7A9R4_ONE_COMMON_CONVENTION_EXISTS_628HC7A9R4,
)

print(
    "Common matching convention(s)                        :",
    FD002_C7A9R4_COMMON_HASH_CONVENTIONS_628HC7A9R4,
)

print(
    "\nC7-A9-R4 complete. "
    "No authority or scoring decision is made by this cell."
)

In [ ]:
# STEP 6.28H-C7-A9-R5
# Strict-whitelist historical MC-helper implementation + execution-constant audit
# NO MODEL INFERENCE / NO MC EXECUTION / NO LABEL OR SCORING ACCESS

import inspect
import hashlib
import pandas as pd

# --------------------------------------------------------------------------------------
# Preconditions
# --------------------------------------------------------------------------------------

if not bool(
    globals().get(
        "FD002_C7A9R4_ALL_ARTIFACTS_HAVE_MATCH_628HC7A9R4",
        False,
    )
):
    raise RuntimeError(
        "C7-A9-R4 must establish exact historical pre-label artifact hashes "
        "before C7-A9-R5."
    )

# --------------------------------------------------------------------------------------
# Strict whitelists only
# --------------------------------------------------------------------------------------

C7A9R5_HELPER_WHITELIST = [
    "mc_dropout_predict_628far2",
    "mc_dropout_predict_628fbr1",
    "mc_dropout_predict_628hc0",
]

C7A9R5_SCALAR_WHITELIST = [
    "FD002_SPLIT_SEED_628B",
    "FD002_REGIME_SEED_628C",

    # Both transferred spellings are deliberately checked.
    "FD002_TRAINING_SEED_628EA",
    "FD002_TRAINING_SEED_628FA",

    "DET_BATCH_628FBR1",
    "MC_BATCH_628FBR1",
]

# --------------------------------------------------------------------------------------
# Helper structural audit — definition inspection only
# --------------------------------------------------------------------------------------

def _helper_code_fingerprint_628hc7a9r5(fn):
    code = fn.__code__

    primitive_types = (
        str,
        int,
        float,
        bool,
        bytes,
        type(None),
    )

    safe_consts = tuple(
        const
        if isinstance(const, primitive_types)
        else f"<{type(const).__name__}>"
        for const in code.co_consts
    )

    payload_parts = [
        code.co_code,
        repr(code.co_names).encode("utf-8"),
        repr(code.co_varnames).encode("utf-8"),
        repr(safe_consts).encode("utf-8"),
        repr(code.co_argcount).encode("utf-8"),
        repr(code.co_kwonlyargcount).encode("utf-8"),
        repr(fn.__defaults__).encode("utf-8"),
        repr(fn.__kwdefaults__).encode("utf-8"),
    ]

    h = hashlib.sha256()

    for part in payload_parts:
        h.update(part)
        h.update(b"\0")

    return h.hexdigest()


helper_rows_628hc7a9r5 = []
helper_sources_628hc7a9r5 = {}

for name_628hc7a9r5 in C7A9R5_HELPER_WHITELIST:

    fn_628hc7a9r5 = globals().get(
        name_628hc7a9r5,
        None,
    )

    if fn_628hc7a9r5 is None:

        helper_rows_628hc7a9r5.append(
            {
                "name": name_628hc7a9r5,
                "present": False,
                "callable": False,
                "signature": None,
                "defaults": None,
                "code_fingerprint_sha256": None,
                "source_available": False,
                "source_sha256": None,
                "referenced_global_names": None,
            }
        )

        helper_sources_628hc7a9r5[
            name_628hc7a9r5
        ] = "<missing>"

        continue

    is_callable_628hc7a9r5 = callable(
        fn_628hc7a9r5
    )

    if not is_callable_628hc7a9r5:

        helper_rows_628hc7a9r5.append(
            {
                "name": name_628hc7a9r5,
                "present": True,
                "callable": False,
                "signature": None,
                "defaults": None,
                "code_fingerprint_sha256": None,
                "source_available": False,
                "source_sha256": None,
                "referenced_global_names": None,
            }
        )

        helper_sources_628hc7a9r5[
            name_628hc7a9r5
        ] = f"<not callable: {type(fn_628hc7a9r5).__name__}>"

        continue

    try:
        signature_628hc7a9r5 = str(
            inspect.signature(
                fn_628hc7a9r5
            )
        )
    except (TypeError, ValueError):
        signature_628hc7a9r5 = "<signature unavailable>"

    try:
        source_628hc7a9r5 = inspect.getsource(
            fn_628hc7a9r5
        )

        source_available_628hc7a9r5 = True

        source_sha_628hc7a9r5 = hashlib.sha256(
            source_628hc7a9r5.encode("utf-8")
        ).hexdigest()

    except (OSError, TypeError):

        source_628hc7a9r5 = "<source unavailable>"
        source_available_628hc7a9r5 = False
        source_sha_628hc7a9r5 = None

    code_fingerprint_628hc7a9r5 = None
    referenced_names_628hc7a9r5 = None

    if hasattr(fn_628hc7a9r5, "__code__"):

        code_fingerprint_628hc7a9r5 = (
            _helper_code_fingerprint_628hc7a9r5(
                fn_628hc7a9r5
            )
        )

        referenced_names_628hc7a9r5 = list(
            fn_628hc7a9r5.__code__.co_names
        )

    helper_rows_628hc7a9r5.append(
        {
            "name": name_628hc7a9r5,
            "present": True,
            "callable": True,
            "signature": signature_628hc7a9r5,
            "defaults": repr(
                fn_628hc7a9r5.__defaults__
            ),
            "code_fingerprint_sha256":
                code_fingerprint_628hc7a9r5,
            "source_available":
                source_available_628hc7a9r5,
            "source_sha256":
                source_sha_628hc7a9r5,
            "referenced_global_names":
                referenced_names_628hc7a9r5,
        }
    )

    helper_sources_628hc7a9r5[
        name_628hc7a9r5
    ] = source_628hc7a9r5


FD002_C7A9R5_HELPER_AUDIT_628HC7A9R5 = pd.DataFrame(
    helper_rows_628hc7a9r5
)

# --------------------------------------------------------------------------------------
# Exact historical scalar inspection
# --------------------------------------------------------------------------------------

scalar_rows_628hc7a9r5 = []

for name_628hc7a9r5 in C7A9R5_SCALAR_WHITELIST:

    present_628hc7a9r5 = (
        name_628hc7a9r5 in globals()
    )

    value_628hc7a9r5 = (
        globals()[name_628hc7a9r5]
        if present_628hc7a9r5
        else None
    )

    if (
        present_628hc7a9r5
        and
        isinstance(
            value_628hc7a9r5,
            (str, int, float, bool, type(None)),
        )
    ):
        safe_value_628hc7a9r5 = repr(
            value_628hc7a9r5
        )
    elif present_628hc7a9r5:
        safe_value_628hc7a9r5 = (
            f"<{type(value_628hc7a9r5).__name__}: not expanded>"
        )
    else:
        safe_value_628hc7a9r5 = "<missing>"

    scalar_rows_628hc7a9r5.append(
        {
            "name": name_628hc7a9r5,
            "present": present_628hc7a9r5,
            "type": (
                type(value_628hc7a9r5).__name__
                if present_628hc7a9r5
                else None
            ),
            "value": safe_value_628hc7a9r5,
        }
    )


FD002_C7A9R5_EXECUTION_SCALARS_628HC7A9R5 = pd.DataFrame(
    scalar_rows_628hc7a9r5
)

# --------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.28H-C7-A9-R5 — STRICT-WHITELIST HISTORICAL "
    "MC-HELPER IMPLEMENTATION + EXECUTION-CONSTANT AUDIT"
)
print("=" * 126)

print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("No prediction/risk/mask values inspected.")
print("No broad namespace discovery performed.")
print("No frozen project state modified.")
print("No files written.")

print("\n--- MC HELPER STRUCTURAL AUDIT ---")

print(
    FD002_C7A9R5_HELPER_AUDIT_628HC7A9R5[
        [
            "name",
            "present",
            "callable",
            "signature",
            "defaults",
            "code_fingerprint_sha256",
            "source_available",
            "source_sha256",
        ]
    ].to_string(index=False)
)

print("\n--- HISTORICAL EXECUTION SCALARS ---")

print(
    FD002_C7A9R5_EXECUTION_SCALARS_628HC7A9R5.to_string(
        index=False
    )
)

print("\n--- EXACT WHITELISTED HELPER SOURCES ---")

for name_628hc7a9r5 in C7A9R5_HELPER_WHITELIST:

    print("\n" + "-" * 110)
    print(name_628hc7a9r5)
    print("-" * 110)

    print(
        helper_sources_628hc7a9r5[
            name_628hc7a9r5
        ]
    )

print(
    "\nC7-A9-R5 complete: helper definitions and exact historical "
    "execution constants inspected only; no helper was executed."
)

In [ ]:
# STEP 6.28H-C7-A9-R6
# Historical model-state fingerprint convention probe
# STRICT WHITELIST / NO MODEL INFERENCE / NO LABEL ACCESS

import hashlib
import io
import torch
import numpy as np
import pandas as pd

if "FD002_C7A9R1_MODEL_STATE_GROUPS_628HC7A9R1" not in globals():
    raise RuntimeError(
        "C7-A9-R1 model-state grouping is required before C7-A9-R6."
    )

required_models_628hc7a9r6 = [
    "MODEL_FD002_628EB",
    "model",
]

for name_628hc7a9r6 in required_models_628hc7a9r6:
    if name_628hc7a9r6 not in globals():
        raise RuntimeError(
            f"Required whitelisted model object is missing: "
            f"{name_628hc7a9r6}"
        )

    if not isinstance(
        globals()[name_628hc7a9r6],
        torch.nn.Module,
    ):
        raise RuntimeError(
            f"{name_628hc7a9r6} is not a torch.nn.Module."
        )


def _sha256_628hc7a9r6(payload):
    return hashlib.sha256(payload).hexdigest()


def _state_hash_candidates_628hc7a9r6(model):

    state_items = sorted(
        model.state_dict().items(),
        key=lambda kv: kv[0],
    )

    # ----------------------------------------------------------
    # Convention A:
    # exact convention already used by C7-A9 / R1
    # ----------------------------------------------------------

    h_existing = hashlib.sha256()

    for key, tensor in state_items:

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h_existing.update(
            key.encode("utf-8")
        )

        h_existing.update(
            str(arr.dtype).encode("utf-8")
        )

        h_existing.update(
            np.asarray(
                arr.shape,
                dtype=np.int64,
            ).tobytes()
        )

        h_existing.update(
            arr.tobytes(order="C")
        )

    # ----------------------------------------------------------
    # Convention B:
    # tensor bytes only, sorted state_dict order
    # ----------------------------------------------------------

    h_raw = hashlib.sha256()

    for _, tensor in state_items:

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h_raw.update(
            arr.tobytes(order="C")
        )

    # ----------------------------------------------------------
    # Convention C:
    # key + raw tensor bytes
    # ----------------------------------------------------------

    h_key_raw = hashlib.sha256()

    for key, tensor in state_items:

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h_key_raw.update(
            key.encode("utf-8")
        )

        h_key_raw.update(
            arr.tobytes(order="C")
        )

    # ----------------------------------------------------------
    # Convention D:
    # key + dtype text + raw bytes
    # ----------------------------------------------------------

    h_key_dtype_raw = hashlib.sha256()

    for key, tensor in state_items:

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        h_key_dtype_raw.update(
            key.encode("utf-8")
        )

        h_key_dtype_raw.update(
            str(arr.dtype).encode("utf-8")
        )

        h_key_dtype_raw.update(
            arr.tobytes(order="C")
        )

    # ----------------------------------------------------------
    # Convention E:
    # deterministic concatenation of per-tensor SHA hashes
    # ----------------------------------------------------------

    h_per_tensor = hashlib.sha256()

    for key, tensor in state_items:

        arr = (
            tensor
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        tensor_sha = hashlib.sha256(
            arr.tobytes(order="C")
        ).hexdigest()

        h_per_tensor.update(
            key.encode("utf-8")
        )

        h_per_tensor.update(
            tensor_sha.encode("utf-8")
        )

    # ----------------------------------------------------------
    # Convention F:
    # torch.save(state_dict) serialization in-memory
    # Diagnostic only; no file is written.
    # ----------------------------------------------------------

    buffer_628hc7a9r6 = io.BytesIO()

    torch.save(
        model.state_dict(),
        buffer_628hc7a9r6,
    )

    torch_save_sha = _sha256_628hc7a9r6(
        buffer_628hc7a9r6.getvalue()
    )

    return {
        "existing_C7A9_convention":
            h_existing.hexdigest(),

        "raw_tensor_bytes_only":
            h_raw.hexdigest(),

        "key_plus_raw":
            h_key_raw.hexdigest(),

        "key_dtype_plus_raw":
            h_key_dtype_raw.hexdigest(),

        "key_plus_per_tensor_sha":
            h_per_tensor.hexdigest(),

        "torch_save_state_dict_bytes":
            torch_save_sha,
    }


rows_628hc7a9r6 = []

for name_628hc7a9r6 in required_models_628hc7a9r6:

    model_628hc7a9r6 = globals()[
        name_628hc7a9r6
    ]

    candidates_628hc7a9r6 = (
        _state_hash_candidates_628hc7a9r6(
            model_628hc7a9r6
        )
    )

    for convention_628hc7a9r6, sha_628hc7a9r6 in (
        candidates_628hc7a9r6.items()
    ):

        rows_628hc7a9r6.append(
            {
                "model_object":
                    name_628hc7a9r6,

                "convention":
                    convention_628hc7a9r6,

                "sha256":
                    sha_628hc7a9r6,

                "starts_ba9376":
                    sha_628hc7a9r6.startswith(
                        "ba9376"
                    ),

                "starts_024d79":
                    sha_628hc7a9r6.startswith(
                        "024d79"
                    ),

                "starts_e91327":
                    sha_628hc7a9r6.startswith(
                        "e91327"
                    ),
            }
        )


FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6 = (
    pd.DataFrame(
        rows_628hc7a9r6
    )
)

print("=" * 122)

print(
    "STEP 6.28H-C7-A9-R6 — HISTORICAL MODEL-STATE "
    "FINGERPRINT CONVENTION PROBE"
)

print("=" * 122)

print("No model forward pass executed.")
print("No MC-dropout execution performed.")
print("No RUL/label/scoring/error/MAE object inspected.")
print("Only two already-whitelisted model state_dicts were hashed.")
print("No frozen project state modified.")
print("No files written.")

print()

print(
    FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6
    .to_string(index=False)
)

print("\n--- PREFIX SUMMARY ---")

for name_628hc7a9r6 in required_models_628hc7a9r6:

    subset_628hc7a9r6 = (
        FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6[
            FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6[
                "model_object"
            ]
            ==
            name_628hc7a9r6
        ]
    )

    print(
        name_628hc7a9r6,
        "| ba9376 match:",
        bool(
            subset_628hc7a9r6[
                "starts_ba9376"
            ].any()
        ),
        "| 024d79 match:",
        bool(
            subset_628hc7a9r6[
                "starts_024d79"
            ].any()
        ),
        "| e91327 match:",
        bool(
            subset_628hc7a9r6[
                "starts_e91327"
            ].any()
        ),
    )

print(
    "\nC7-A9-R6 complete. "
    "This cell only investigates the transferred historical "
    "model-hash discrepancy; it does not assign authority or "
    "perform inference."
)

In [ ]:
# STEP 6.28H-C7-A9-CLOSE
# FD002 authoritative historical pre-label execution reuse-path closure
# NO INFERENCE / NO LABEL ACCESS / NO SCORING / NO RETUNING

import hashlib
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# Preconditions from the corrective C7-A9 recovery sequence
# --------------------------------------------------------------------------------------

required_names_628hc7a9c = [
    "FD002_C7A9R1_MODEL_STATE_GROUPS_628HC7A9R1",
    "FD002_C7A9R4_ALL_ARTIFACTS_HAVE_MATCH_628HC7A9R4",
    "FD002_C7A9R5_HELPER_AUDIT_628HC7A9R5",
    "FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
    "FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3",
    "FD002_C5_RELOADED_METADATA_628HC5F1B",
    "metadata_628hc5f2a",
    "FD002_CANONICAL_TEST_PREDICTION_628HC5F2A",
    "FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2",
    "FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B",
    "FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B",
    "TEST_MC_MATRIX_FD002_628FAR2",
    "MODEL_FD002_628EB",
    "model",
]

missing_628hc7a9c = [
    name
    for name in required_names_628hc7a9c
    if name not in globals()
]

if missing_628hc7a9c:
    raise RuntimeError(
        "Missing required C7-A9 closure object(s): "
        + ", ".join(missing_628hc7a9c)
    )

if not bool(FD002_C7A9R4_ALL_ARTIFACTS_HAVE_MATCH_628HC7A9R4):
    raise RuntimeError(
        "R4 did not cryptographically recover the historical pre-label artifacts."
    )

# --------------------------------------------------------------------------------------
# Exact historical pre-label artifact identity
# Historical convention recovered at R4: raw contiguous array bytes.
# --------------------------------------------------------------------------------------

contract_628hc7a9c = FD002_TEST_PRESCORING_CONTRACT_628HC5A5

def _raw_array_sha256_628hc7a9c(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype artifact refused.")

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


artifact_checks_628hc7a9c = {
    "prediction": (
        FD002_CANONICAL_TEST_PREDICTION_628HC5F2A,
        contract_628hc7a9c["prediction_sha256"],
    ),
    "risk": (
        FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2,
        contract_628hc7a9c["risk_sha256"],
    ),
    "accept_mask": (
        FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B,
        contract_628hc7a9c["accept_mask_sha256"],
    ),
    "abstain_mask": (
        FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B,
        contract_628hc7a9c["abstain_mask_sha256"],
    ),
}

artifact_rows_628hc7a9c = []

for artifact_name_628hc7a9c, (
    artifact_628hc7a9c,
    expected_sha_628hc7a9c,
) in artifact_checks_628hc7a9c.items():

    actual_sha_628hc7a9c = _raw_array_sha256_628hc7a9c(
        artifact_628hc7a9c
    )

    artifact_rows_628hc7a9c.append(
        {
            "artifact": artifact_name_628hc7a9c,
            "historical_sha256": str(expected_sha_628hc7a9c),
            "current_sha256": actual_sha_628hc7a9c,
            "exact_historical_identity": bool(
                actual_sha_628hc7a9c
                == str(expected_sha_628hc7a9c)
            ),
        }
    )

FD002_C7A9C_ARTIFACT_IDENTITY_628HC7A9C = pd.DataFrame(
    artifact_rows_628hc7a9c
)

historical_artifacts_exact_628hc7a9c = bool(
    FD002_C7A9C_ARTIFACT_IDENTITY_628HC7A9C[
        "exact_historical_identity"
    ].all()
)

# --------------------------------------------------------------------------------------
# Historical pre-label status flags
# --------------------------------------------------------------------------------------

policy_meta_628hc7a9c = FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3
reload_meta_628hc7a9c = FD002_C5_RELOADED_METADATA_628HC5F1B
pred_meta_628hc7a9c = metadata_628hc5f2a

prelabel_flags_ok_628hc7a9c = bool(
    policy_meta_628hc7a9c.get("official_test_labels_read") is False
    and policy_meta_628hc7a9c.get("test_scored") is False
    and reload_meta_628hc7a9c.get("official_test_labels_read") is False
    and reload_meta_628hc7a9c.get("test_scored") is False
    and reload_meta_628hc7a9c.get("threshold_tuned_or_modified") is False
    and pred_meta_628hc7a9c.get("official_test_labels_read") is False
    and pred_meta_628hc7a9c.get("test_scored") is False
    and pred_meta_628hc7a9c.get("threshold_tuned_or_modified") is False
)

# --------------------------------------------------------------------------------------
# MC historical execution lineage
# --------------------------------------------------------------------------------------

mc_matrix_628hc7a9c = np.asarray(
    TEST_MC_MATRIX_FD002_628FAR2
)

mc_matrix_contract_ok_628hc7a9c = bool(
    mc_matrix_628hc7a9c.shape == (50, 259)
    and mc_matrix_628hc7a9c.dtype == np.float32
    and np.isfinite(mc_matrix_628hc7a9c).all()
)

helper_audit_628hc7a9c = FD002_C7A9R5_HELPER_AUDIT_628HC7A9R5

far2_row_628hc7a9c = helper_audit_628hc7a9r5 = helper_audit_628hc7a9c[
    helper_audit_628hc7a9c["name"]
    == "mc_dropout_predict_628far2"
]

fbr1_row_628hc7a9c = helper_audit_628hc7a9c[
    helper_audit_628hc7a9c["name"]
    == "mc_dropout_predict_628fbr1"
]

helper_lineage_supported_628hc7a9c = bool(
    len(far2_row_628hc7a9c) == 1
    and len(fbr1_row_628hc7a9c) == 1
    and bool(far2_row_628hc7a9c.iloc[0]["present"])
    and bool(far2_row_628hc7a9c.iloc[0]["callable"])
    and (
        far2_row_628hc7a9c.iloc[0]["code_fingerprint_sha256"]
        ==
        fbr1_row_628hc7a9c.iloc[0]["code_fingerprint_sha256"]
    )
)

FD002_C7A9C_HISTORICAL_MC_LINEAGE_628HC7A9C = (
    "628FAR2 direct artifact lineage; "
    "mc_dropout_predict_628far2 is bytecode-equivalent to 628FBR1"
)

# --------------------------------------------------------------------------------------
# Resident model-state provenance limitation
# --------------------------------------------------------------------------------------

model_hash_audit_628hc7a9c = (
    FD002_C7A9R6_MODEL_HASH_CONVENTION_AUDIT_628HC7A9R6
)

fd002_hash_rows_628hc7a9c = model_hash_audit_628hc7a9c[
    model_hash_audit_628hc7a9c["model_object"]
    == "MODEL_FD002_628EB"
]

later_model_hash_rows_628hc7a9c = model_hash_audit_628hc7a9c[
    model_hash_audit_628hc7a9c["model_object"]
    == "model"
]

resident_fd002_matches_transferred_ba9376_628hc7a9c = bool(
    fd002_hash_rows_628hc7a9c["starts_ba9376"].any()
)

later_model_matches_transferred_024d79_628hc7a9c = bool(
    later_model_hash_rows_628hc7a9c["starts_024d79"].any()
)

resident_fd002_current_sha_628hc7a9c = str(
    fd002_hash_rows_628hc7a9c.loc[
        fd002_hash_rows_628hc7a9c["convention"]
        == "existing_C7A9_convention",
        "sha256",
    ].iloc[0]
)

resident_model_historical_identity_proven_628hc7a9c = bool(
    resident_fd002_matches_transferred_ba9376_628hc7a9c
)

FD002_C7A9C_RESIDENT_MODEL_PROVENANCE_NOTE_628HC7A9C = (
    "Current MODEL_FD002_628EB belongs to the distinct FD002-named alias group, "
    "but its currently resident state hash is "
    f"{resident_fd002_current_sha_628hc7a9c}; "
    "none of the tested deterministic conventions reproduced the transferred "
    "historical ba9376... FD002 lineage. Therefore the current resident weights "
    "are NOT claimed to be the exact original historical TEST-execution state. "
    "This does not invalidate the recovered TEST execution because the historical "
    "pre-label prediction/risk/decision artifacts are independently hash-verified."
)

# --------------------------------------------------------------------------------------
# Reuse-path decision
# --------------------------------------------------------------------------------------

reuse_historical_execution_628hc7a9c = bool(
    historical_artifacts_exact_628hc7a9c
    and prelabel_flags_ok_628hc7a9c
    and mc_matrix_contract_ok_628hc7a9c
    and helper_lineage_supported_628hc7a9c
)

new_inference_required_628hc7a9c = bool(
    not reuse_historical_execution_628hc7a9c
)

if new_inference_required_628hc7a9c:
    raise RuntimeError(
        "Historical pre-label execution could not be fully verified. "
        "STOP: do not admit official RUL labels."
    )

# --------------------------------------------------------------------------------------
# Final corrected C7-A9 closure audit
# --------------------------------------------------------------------------------------

audit_628hc7a9c = pd.DataFrame(
    {
        "check": [
            "R4_historical_prediction_risk_masks_exact",
            "historical_prelabel_flags_consistent",
            "historical_MC_matrix_contract_50x259_float32_finite",
            "628FAR2_helper_lineage_supported",
            "later_FD001_state_reproduces_transferred_024d79_prefix",
            "resident_FD002_state_reproduces_transferred_ba9376_prefix",
            "resident_model_exact_historical_state_proven",
            "historical_prelabel_execution_reusable",
            "new_model_or_MC_inference_required",
            "official_RUL_admission_performed",
            "TEST_scoring_performed",
            "DEV_retuning_performed",
            "frozen_policy_modified",
        ],
        "status": [
            historical_artifacts_exact_628hc7a9c,
            prelabel_flags_ok_628hc7a9c,
            mc_matrix_contract_ok_628hc7a9c,
            helper_lineage_supported_628hc7a9c,
            later_model_matches_transferred_024d79_628hc7a9c,
            resident_fd002_matches_transferred_ba9376_628hc7a9c,
            resident_model_historical_identity_proven_628hc7a9c,
            reuse_historical_execution_628hc7a9c,
            new_inference_required_628hc7a9c,
            False,
            False,
            False,
            False,
        ],
        "interpretation": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "KNOWN LIMITATION — does not invalidate verified historical outputs",
            "KNOWN LIMITATION — current resident weights not used for rerun",
            "PASS",
            "FALSE AS REQUIRED",
            "FALSE AS REQUIRED",
            "FALSE AS REQUIRED",
            "FALSE AS REQUIRED",
            "FALSE AS REQUIRED",
        ],
    }
)

FD002_C7A9_FINAL_AUDIT_628HC7A9C = audit_628hc7a9c

FD002_C7A9_FINAL_STATUS_628HC7A9C = (
    "PASS_REUSE_HISTORICAL_PRELABEL_EXECUTION_WITH_RESIDENT_MODEL_STATE_LIMITATION"
)

FD002_C7A9_REUSE_HISTORICAL_EXECUTION_628HC7A9C = True
FD002_C7A9_NEW_INFERENCE_REQUIRED_628HC7A9C = False

FD002_C7A9_RESIDENT_MODEL_HISTORICAL_IDENTITY_PROVEN_628HC7A9C = (
    resident_model_historical_identity_proven_628hc7a9c
)

all_checks_passed_628hc7a9c = True

# --------------------------------------------------------------------------------------
# Output
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A9 — CORRECTED FINAL CLOSURE "
    "(VERIFY-AND-REUSE PATH)"
)
print("=" * 128)

print("\n--- EXACT HISTORICAL PRE-LABEL ARTIFACT IDENTITY ---")
print(
    FD002_C7A9C_ARTIFACT_IDENTITY_628HC7A9C.to_string(
        index=False
    )
)

print("\n--- CORRECTED C7-A9 FINAL AUDIT ---")
print(
    FD002_C7A9_FINAL_AUDIT_628HC7A9C.to_string(
        index=False
    )
)

print("\n--- MODEL PROVENANCE NOTE ---")
print(
    FD002_C7A9C_RESIDENT_MODEL_PROVENANCE_NOTE_628HC7A9C
)

print("\n--- FINAL C7-A9 DECISION ---")
print("Status                         :", FD002_C7A9_FINAL_STATUS_628HC7A9C)
print(
    "Historical execution reusable  :",
    FD002_C7A9_REUSE_HISTORICAL_EXECUTION_628HC7A9C,
)
print(
    "New model/MC inference required :",
    FD002_C7A9_NEW_INFERENCE_REQUIRED_628HC7A9C,
)
print(
    "Historical MC lineage           :",
    FD002_C7A9C_HISTORICAL_MC_LINEAGE_628HC7A9C,
)
print("Official RUL admitted           :", False)
print("TEST scored                     :", False)
print("DEV retuned                     :", False)
print("Frozen policy modified          :", False)

print(
    "\nPASS — C7-A9 closes on the verify-and-reuse path. "
    "The exact historical pre-label TEST prediction, uncertainty-risk and "
    "accept/abstain decisions are cryptographically verified and reusable. "
    "No new deterministic or MC inference is justified."
)

print(
    "\nIMPORTANT LIMITATION — the currently resident FD002-named model weights "
    "are not claimed to be the exact original historical TEST-execution weights. "
    "They will not be used to regenerate the already-verified historical outputs."
)

print(
    "\nNext stage: STEP 6.28H-C7-A10 — freeze the verified historical "
    "pre-label TEST prediction / MC uncertainty / risk / decisions for the "
    "current one-way evaluation chain, still before any official RUL admission."
)

In [ ]:
# STEP 6.28H-C7-A10
# FD002 verified historical PRE-LABEL execution freeze for the one-way evaluation chain
# NO RUL ACCESS / NO SCORING / NO MODEL OR MC INFERENCE / NO RETUNING

import hashlib
import json
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions: C7-A9 must be closed on the verify-and-reuse path
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a9c", False)):
    raise RuntimeError(
        "Corrected C7-A9 closure has not passed. Do not continue to C7-A10."
    )

if not bool(
    globals().get(
        "FD002_C7A9_REUSE_HISTORICAL_EXECUTION_628HC7A9C",
        False,
    )
):
    raise RuntimeError(
        "C7-A9 did not authorize historical pre-label execution reuse."
    )

if bool(
    globals().get(
        "FD002_C7A9_NEW_INFERENCE_REQUIRED_628HC7A9C",
        True,
    )
):
    raise RuntimeError(
        "C7-A9 indicates new inference is required. Stop before C7-A10."
    )

required_628hc7a10 = [
    "FD002_CANONICAL_TEST_PREDICTION_628HC5F2A",
    "TEST_MC_MATRIX_FD002_628FAR2",
    "TEST_MC_STD_FD002_628FAR2",
    "FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2",
    "FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B",
    "FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
    "FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3",
    "FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B",
    "FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7",
]

missing_628hc7a10 = [
    name
    for name in required_628hc7a10
    if name not in globals()
]

if missing_628hc7a10:
    raise RuntimeError(
        "Missing required pre-label freeze object(s): "
        + ", ".join(missing_628hc7a10)
    )

# --------------------------------------------------------------------------------------
# 2) Strictly whitelisted source objects
# --------------------------------------------------------------------------------------

prediction_src_628hc7a10 = np.asarray(
    FD002_CANONICAL_TEST_PREDICTION_628HC5F2A
)

mc_matrix_src_628hc7a10 = np.asarray(
    TEST_MC_MATRIX_FD002_628FAR2
)

mc_std_historical_src_628hc7a10 = np.asarray(
    TEST_MC_STD_FD002_628FAR2
)

risk_src_628hc7a10 = np.asarray(
    FD002_CANONICAL_TEST_UNCERTAINTY_RISK_628HC5A2
)

accept_src_628hc7a10 = np.asarray(
    FD002_C5_RELOADED_ACCEPT_MASK_628HC5F1B
)

abstain_src_628hc7a10 = np.asarray(
    FD002_C5_RELOADED_ABSTAIN_MASK_628HC5F1B
)

prescoring_contract_628hc7a10 = (
    FD002_TEST_PRESCORING_CONTRACT_628HC5A5
)

policy_meta_628hc7a10 = (
    FD002_TEST_POLICY_APPLICATION_METADATA_628HC5A3
)

threshold_628hc7a10 = float(
    FD002_FROZEN_C4_SELECTED_RISK_THRESHOLD_628HC4F1B
)

endpoint_meta_src_628hc7a10 = (
    FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7
)

if not isinstance(endpoint_meta_src_628hc7a10, pd.DataFrame):
    raise RuntimeError(
        "FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7 "
        "is not a pandas DataFrame."
    )

# --------------------------------------------------------------------------------------
# 3) Contract checks before creating the current-chain freeze
# --------------------------------------------------------------------------------------

if not (
    prediction_src_628hc7a10.shape == (259,)
    and np.isfinite(prediction_src_628hc7a10).all()
):
    raise RuntimeError("Prediction contract failed.")

if not (
    mc_matrix_src_628hc7a10.shape == (50, 259)
    and mc_matrix_src_628hc7a10.dtype == np.float32
    and np.isfinite(mc_matrix_src_628hc7a10).all()
):
    raise RuntimeError("Historical MC matrix contract failed.")

if not (
    mc_std_historical_src_628hc7a10.shape == (259,)
    and np.isfinite(mc_std_historical_src_628hc7a10).all()
):
    raise RuntimeError("Historical MC STD contract failed.")

if not (
    risk_src_628hc7a10.shape == (259,)
    and np.isfinite(risk_src_628hc7a10).all()
    and ((risk_src_628hc7a10 >= 0.0) & (risk_src_628hc7a10 <= 1.0)).all()
):
    raise RuntimeError("Reliability-risk contract failed.")

if not (
    accept_src_628hc7a10.shape == (259,)
    and abstain_src_628hc7a10.shape == (259,)
):
    raise RuntimeError("Decision-mask shape contract failed.")

accept_bool_628hc7a10 = accept_src_628hc7a10.astype(
    bool,
    copy=False,
)

abstain_bool_628hc7a10 = abstain_src_628hc7a10.astype(
    bool,
    copy=False,
)

expected_accept_628hc7a10 = (
    risk_src_628hc7a10
    <=
    threshold_628hc7a10
)

if not np.array_equal(
    expected_accept_628hc7a10,
    accept_bool_628hc7a10,
):
    raise RuntimeError(
        "Frozen accept mask does not exactly satisfy risk <= threshold."
    )

if not np.array_equal(
    abstain_bool_628hc7a10,
    ~accept_bool_628hc7a10,
):
    raise RuntimeError(
        "Frozen abstain mask is not the exact complement of accept."
    )

if len(endpoint_meta_src_628hc7a10) != 259:
    raise RuntimeError(
        "Locked TEST endpoint metadata does not contain 259 rows."
    )

# Historical helper semantics recovered in R5:
# float32 MC matrix -> mean/std on that matrix -> cast outputs to float64.
mc_mean_derived_628hc7a10 = (
    mc_matrix_src_628hc7a10
    .mean(axis=0)
    .astype(np.float64)
)

mc_std_derived_628hc7a10 = (
    mc_matrix_src_628hc7a10
    .std(axis=0, ddof=0)
    .astype(np.float64)
)

mc_std_exact_historical_628hc7a10 = bool(
    np.array_equal(
        mc_std_derived_628hc7a10,
        np.asarray(
            mc_std_historical_src_628hc7a10,
            dtype=np.float64,
        ),
    )
)

if not mc_std_exact_historical_628hc7a10:
    raise RuntimeError(
        "Derived MC STD is not exactly identical to "
        "TEST_MC_STD_FD002_628FAR2. Stop before freezing."
    )

if float(prescoring_contract_628hc7a10["frozen_risk_threshold"]) != threshold_628hc7a10:
    raise RuntimeError(
        "Pre-scoring contract threshold differs from frozen C4 threshold."
    )

if str(policy_meta_628hc7a10.get("reliability_architecture")) != "uncertainty_only":
    raise RuntimeError(
        "Unexpected reliability architecture in pre-label metadata."
    )

if str(policy_meta_628hc7a10.get("operating_policy")) != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Unexpected operating policy in pre-label metadata."
    )

if policy_meta_628hc7a10.get("official_test_labels_read") is not False:
    raise RuntimeError(
        "Pre-label metadata does not certify official_test_labels_read=False."
    )

if policy_meta_628hc7a10.get("test_scored") is not False:
    raise RuntimeError(
        "Pre-label metadata does not certify test_scored=False."
    )

# --------------------------------------------------------------------------------------
# 4) Historical hash check using the exact convention recovered in R4
# --------------------------------------------------------------------------------------

def _raw_sha256_628hc7a10(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused for raw hashing.")

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


historical_hash_checks_628hc7a10 = {
    "prediction": (
        prediction_src_628hc7a10,
        prescoring_contract_628hc7a10["prediction_sha256"],
    ),
    "risk": (
        risk_src_628hc7a10,
        prescoring_contract_628hc7a10["risk_sha256"],
    ),
    "accept_mask": (
        accept_bool_628hc7a10,
        prescoring_contract_628hc7a10["accept_mask_sha256"],
    ),
    "abstain_mask": (
        abstain_bool_628hc7a10,
        prescoring_contract_628hc7a10["abstain_mask_sha256"],
    ),
}

for artifact_name_628hc7a10, (
    artifact_value_628hc7a10,
    expected_sha_628hc7a10,
) in historical_hash_checks_628hc7a10.items():

    actual_sha_628hc7a10 = _raw_sha256_628hc7a10(
        artifact_value_628hc7a10
    )

    if actual_sha_628hc7a10 != str(expected_sha_628hc7a10):
        raise RuntimeError(
            f"Historical identity failed for {artifact_name_628hc7a10}."
        )

# --------------------------------------------------------------------------------------
# 5) Create CURRENT-CHAIN frozen copies
# --------------------------------------------------------------------------------------

FD002_C7A10_FROZEN_PREDICTION_628HC7A10 = np.array(
    prediction_src_628hc7a10,
    copy=True,
)

FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10 = np.array(
    mc_matrix_src_628hc7a10,
    copy=True,
)

FD002_C7A10_FROZEN_MC_MEAN_628HC7A10 = np.array(
    mc_mean_derived_628hc7a10,
    copy=True,
)

FD002_C7A10_FROZEN_MC_STD_628HC7A10 = np.array(
    mc_std_derived_628hc7a10,
    copy=True,
)

FD002_C7A10_FROZEN_RISK_628HC7A10 = np.array(
    risk_src_628hc7a10,
    copy=True,
)

FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10 = np.array(
    accept_bool_628hc7a10,
    dtype=bool,
    copy=True,
)

FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10 = np.array(
    abstain_bool_628hc7a10,
    dtype=bool,
    copy=True,
)

# Make NumPy freeze objects read-only in the current session.
for arr_628hc7a10 in [
    FD002_C7A10_FROZEN_PREDICTION_628HC7A10,
    FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10,
    FD002_C7A10_FROZEN_MC_MEAN_628HC7A10,
    FD002_C7A10_FROZEN_MC_STD_628HC7A10,
    FD002_C7A10_FROZEN_RISK_628HC7A10,
    FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10,
    FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10,
]:
    arr_628hc7a10.setflags(write=False)

FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10 = (
    endpoint_meta_src_628hc7a10.copy(deep=True)
)

FD002_C7A10_FROZEN_THRESHOLD_628HC7A10 = float(
    threshold_628hc7a10
)

FD002_C7A10_FROZEN_ARCHITECTURE_628HC7A10 = (
    "uncertainty_only"
)

FD002_C7A10_FROZEN_OPERATING_POLICY_628HC7A10 = (
    "uncertainty_only_fixed_threshold"
)

FD002_C7A10_FROZEN_COMPARATOR_628HC7A10 = "<="

FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10 = 0.80

# --------------------------------------------------------------------------------------
# 6) Current-chain fingerprints for the next C7-A11 audit
# --------------------------------------------------------------------------------------

def _metadata_sha256_628hc7a10(df):
    payload = df.to_csv(
        index=True,
        lineterminator="\n",
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


endpoint_metadata_sha_628hc7a10 = (
    _metadata_sha256_628hc7a10(
        FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10
    )
)

FD002_C7A10_PRELABEL_FINGERPRINTS_628HC7A10 = {
    "prediction_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_PREDICTION_628HC7A10
        ),

    "mc_matrix_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10
        ),

    "mc_mean_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_MC_MEAN_628HC7A10
        ),

    "mc_std_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_MC_STD_628HC7A10
        ),

    "risk_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_RISK_628HC7A10
        ),

    "accept_mask_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
        ),

    "abstain_mask_sha256":
        _raw_sha256_628hc7a10(
            FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
        ),

    "endpoint_metadata_sha256":
        endpoint_metadata_sha_628hc7a10,
}

accepted_count_628hc7a10 = int(
    FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10.sum()
)

abstained_count_628hc7a10 = int(
    FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10.sum()
)

FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10 = {
    "step": "6.28H-C7-A10",
    "dataset": "FD002",
    "freeze_path": "verified_historical_prelabel_execution_reuse",
    "official_test_labels_admitted": False,
    "test_scored": False,
    "model_or_mc_rerun_performed": False,
    "reliability_architecture":
        FD002_C7A10_FROZEN_ARCHITECTURE_628HC7A10,
    "operating_policy":
        FD002_C7A10_FROZEN_OPERATING_POLICY_628HC7A10,
    "comparator":
        FD002_C7A10_FROZEN_COMPARATOR_628HC7A10,
    "frozen_risk_threshold":
        FD002_C7A10_FROZEN_THRESHOLD_628HC7A10,
    "nominal_target_coverage":
        FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10,
    "n_test_endpoints": 259,
    "mc_passes": 50,
    "mc_std_ddof": 0,
    "mc_seed": 42,
    "mc_batch_size": 512,
    "accepted_count": accepted_count_628hc7a10,
    "abstained_count": abstained_count_628hc7a10,
    "realized_coverage":
        accepted_count_628hc7a10 / 259.0,
    "historical_mc_lineage":
        "TEST_MC_MATRIX_FD002_628FAR2 / mc_dropout_predict_628far2",
    "resident_model_exact_historical_state_proven": False,
    "resident_model_limitation":
        "Current resident FD002-named weights are not used to regenerate "
        "the independently hash-verified historical pre-label outputs.",
    "fingerprints":
        dict(FD002_C7A10_PRELABEL_FINGERPRINTS_628HC7A10),
}

contract_payload_628hc7a10 = json.dumps(
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10 = (
    hashlib.sha256(
        contract_payload_628hc7a10
    ).hexdigest()
)

all_checks_passed_628hc7a10 = True

# --------------------------------------------------------------------------------------
# 7) Output
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A10 — FD002 VERIFIED HISTORICAL "
    "PRE-LABEL EXECUTION FREEZE"
)
print("=" * 128)

print("No official RUL/label object read.")
print("No TEST scoring performed.")
print("No model forward pass performed.")
print("No MC-dropout execution performed.")
print("No DEV retuning performed.")
print("No frozen policy/threshold modification performed.")
print("No files written.")

print("\n--- FREEZE SUMMARY ---")
print(
    "Prediction shape          :",
    FD002_C7A10_FROZEN_PREDICTION_628HC7A10.shape,
)
print(
    "MC matrix shape / dtype   :",
    FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10.shape,
    "/",
    FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10.dtype,
)
print(
    "MC STD exact historical  :",
    mc_std_exact_historical_628hc7a10,
)
print(
    "Risk shape                :",
    FD002_C7A10_FROZEN_RISK_628HC7A10.shape,
)
print(
    "Threshold / comparator    :",
    FD002_C7A10_FROZEN_THRESHOLD_628HC7A10,
    FD002_C7A10_FROZEN_COMPARATOR_628HC7A10,
)
print(
    "Accepted / abstained      :",
    accepted_count_628hc7a10,
    "/",
    abstained_count_628hc7a10,
)
print(
    "Realized coverage         :",
    accepted_count_628hc7a10 / 259.0,
)
print(
    "Endpoint metadata rows    :",
    len(FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10),
)

print("\n--- CURRENT PRE-LABEL FINGERPRINTS ---")
for key_628hc7a10, value_628hc7a10 in (
    FD002_C7A10_PRELABEL_FINGERPRINTS_628HC7A10.items()
):
    print(
        f"{key_628hc7a10:28s}:",
        value_628hc7a10,
    )

print(
    "\nFreeze-contract SHA-256   :",
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10,
)

print(
    "\nPASS — C7-A10 complete. The verified historical FD002 pre-label "
    "prediction, MC uncertainty, reliability risk, decisions and endpoint "
    "metadata are frozen for the current one-way evaluation chain."
)

print(
    "Official FD002 RUL remains FORBIDDEN until C7-A11 independently "
    "re-verifies these fingerprints and the freeze contract."
)

In [ ]:
# STEP 6.28H-C7-A11
# FD002 independent pre-label fingerprint/hash audit
# NO RUL ACCESS / NO TEST SCORING / NO MODEL OR MC INFERENCE / NO RETUNING

import hashlib
import json
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a10", False)):
    raise RuntimeError(
        "C7-A10 must pass before C7-A11. Official FD002 RUL remains forbidden."
    )

required_628hc7a11 = [
    "FD002_C7A10_FROZEN_PREDICTION_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MEAN_628HC7A10",
    "FD002_C7A10_FROZEN_MC_STD_628HC7A10",
    "FD002_C7A10_FROZEN_RISK_628HC7A10",
    "FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10",
    "FD002_C7A10_FROZEN_THRESHOLD_628HC7A10",
    "FD002_C7A10_FROZEN_ARCHITECTURE_628HC7A10",
    "FD002_C7A10_FROZEN_OPERATING_POLICY_628HC7A10",
    "FD002_C7A10_FROZEN_COMPARATOR_628HC7A10",
    "FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10",
    "FD002_C7A10_PRELABEL_FINGERPRINTS_628HC7A10",
    "FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10",
    "FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10",
    "FD002_TEST_PRESCORING_CONTRACT_628HC5A5",
    "FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7",
]

missing_628hc7a11 = [
    name
    for name in required_628hc7a11
    if name not in globals()
]

if missing_628hc7a11:
    raise RuntimeError(
        "Missing required C7-A11 audit object(s): "
        + ", ".join(missing_628hc7a11)
    )

# --------------------------------------------------------------------------------------
# 2) Independent local fingerprint functions
# --------------------------------------------------------------------------------------

def _raw_array_sha256_628hc7a11(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during C7-A11 fingerprint audit."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _dataframe_sha256_628hc7a11(df):
    if not isinstance(df, pd.DataFrame):
        raise RuntimeError(
            "Endpoint metadata is not a pandas DataFrame."
        )

    payload = df.to_csv(
        index=True,
        lineterminator="\n",
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


# --------------------------------------------------------------------------------------
# 3) Recompute every C7-A10 current-chain fingerprint independently
# --------------------------------------------------------------------------------------

recomputed_fingerprints_628hc7a11 = {
    "prediction_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_PREDICTION_628HC7A10
        ),

    "mc_matrix_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10
        ),

    "mc_mean_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_MC_MEAN_628HC7A10
        ),

    "mc_std_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_MC_STD_628HC7A10
        ),

    "risk_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_RISK_628HC7A10
        ),

    "accept_mask_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
        ),

    "abstain_mask_sha256":
        _raw_array_sha256_628hc7a11(
            FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
        ),

    "endpoint_metadata_sha256":
        _dataframe_sha256_628hc7a11(
            FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10
        ),
}

stored_fingerprints_628hc7a11 = dict(
    FD002_C7A10_PRELABEL_FINGERPRINTS_628HC7A10
)

fingerprint_rows_628hc7a11 = []

for key_628hc7a11 in sorted(recomputed_fingerprints_628hc7a11):
    stored_628hc7a11 = stored_fingerprints_628hc7a11.get(
        key_628hc7a11
    )

    recomputed_628hc7a11 = recomputed_fingerprints_628hc7a11[
        key_628hc7a11
    ]

    fingerprint_rows_628hc7a11.append(
        {
            "artifact": key_628hc7a11,
            "stored_sha256": stored_628hc7a11,
            "recomputed_sha256": recomputed_628hc7a11,
            "exact_match": bool(
                stored_628hc7a11 == recomputed_628hc7a11
            ),
        }
    )

FD002_C7A11_FINGERPRINT_AUDIT_628HC7A11 = pd.DataFrame(
    fingerprint_rows_628hc7a11
)

all_fingerprints_exact_628hc7a11 = bool(
    FD002_C7A11_FINGERPRINT_AUDIT_628HC7A11[
        "exact_match"
    ].all()
)

# --------------------------------------------------------------------------------------
# 4) Recompute the C7-A10 freeze-contract hash independently
# --------------------------------------------------------------------------------------

contract_628hc7a11 = dict(
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10
)

contract_payload_628hc7a11 = json.dumps(
    contract_628hc7a11,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

recomputed_contract_sha_628hc7a11 = hashlib.sha256(
    contract_payload_628hc7a11
).hexdigest()

contract_hash_exact_628hc7a11 = bool(
    recomputed_contract_sha_628hc7a11
    ==
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10
)

# --------------------------------------------------------------------------------------
# 5) Internal execution consistency audit
# --------------------------------------------------------------------------------------

prediction_628hc7a11 = FD002_C7A10_FROZEN_PREDICTION_628HC7A10
mc_matrix_628hc7a11 = FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10
mc_mean_628hc7a11 = FD002_C7A10_FROZEN_MC_MEAN_628HC7A10
mc_std_628hc7a11 = FD002_C7A10_FROZEN_MC_STD_628HC7A10
risk_628hc7a11 = FD002_C7A10_FROZEN_RISK_628HC7A10
accept_628hc7a11 = FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
abstain_628hc7a11 = FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
endpoint_meta_628hc7a11 = FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10

derived_mc_mean_628hc7a11 = (
    mc_matrix_628hc7a11
    .mean(axis=0)
    .astype(np.float64)
)

derived_mc_std_628hc7a11 = (
    mc_matrix_628hc7a11
    .std(axis=0, ddof=0)
    .astype(np.float64)
)

threshold_628hc7a11 = float(
    FD002_C7A10_FROZEN_THRESHOLD_628HC7A10
)

expected_accept_628hc7a11 = (
    risk_628hc7a11
    <=
    threshold_628hc7a11
)

expected_abstain_628hc7a11 = (
    ~expected_accept_628hc7a11
)

arrays_read_only_628hc7a11 = bool(
    all(
        not arr.flags.writeable
        for arr in [
            prediction_628hc7a11,
            mc_matrix_628hc7a11,
            mc_mean_628hc7a11,
            mc_std_628hc7a11,
            risk_628hc7a11,
            accept_628hc7a11,
            abstain_628hc7a11,
        ]
    )
)

endpoint_metadata_exact_source_628hc7a11 = bool(
    endpoint_meta_628hc7a11.equals(
        FD002_C7A7_LOCKED_TEST_ENDPOINT_METADATA_628HC7A7
    )
)

mc_mean_exact_628hc7a11 = bool(
    np.array_equal(
        derived_mc_mean_628hc7a11,
        mc_mean_628hc7a11,
    )
)

mc_std_exact_628hc7a11 = bool(
    np.array_equal(
        derived_mc_std_628hc7a11,
        mc_std_628hc7a11,
    )
)

decision_rule_exact_628hc7a11 = bool(
    np.array_equal(
        expected_accept_628hc7a11,
        accept_628hc7a11,
    )
    and
    np.array_equal(
        expected_abstain_628hc7a11,
        abstain_628hc7a11,
    )
)

counts_exact_628hc7a11 = bool(
    int(accept_628hc7a11.sum()) == 211
    and int(abstain_628hc7a11.sum()) == 48
    and int(accept_628hc7a11.sum())
        + int(abstain_628hc7a11.sum())
        == 259
)

policy_exact_628hc7a11 = bool(
    FD002_C7A10_FROZEN_ARCHITECTURE_628HC7A10
        == "uncertainty_only"
    and FD002_C7A10_FROZEN_OPERATING_POLICY_628HC7A10
        == "uncertainty_only_fixed_threshold"
    and FD002_C7A10_FROZEN_COMPARATOR_628HC7A10
        == "<="
    and threshold_628hc7a11
        == 0.8003345892330226
    and float(
        FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10
    ) == 0.80
)

shape_contract_exact_628hc7a11 = bool(
    prediction_628hc7a11.shape == (259,)
    and mc_matrix_628hc7a11.shape == (50, 259)
    and mc_matrix_628hc7a11.dtype == np.float32
    and mc_mean_628hc7a11.shape == (259,)
    and mc_std_628hc7a11.shape == (259,)
    and risk_628hc7a11.shape == (259,)
    and accept_628hc7a11.shape == (259,)
    and abstain_628hc7a11.shape == (259,)
    and len(endpoint_meta_628hc7a11) == 259
)

finite_contract_exact_628hc7a11 = bool(
    np.isfinite(prediction_628hc7a11).all()
    and np.isfinite(mc_matrix_628hc7a11).all()
    and np.isfinite(mc_mean_628hc7a11).all()
    and np.isfinite(mc_std_628hc7a11).all()
    and np.isfinite(risk_628hc7a11).all()
)

# --------------------------------------------------------------------------------------
# 6) Re-check historical C5 identities for prediction/risk/decisions
# --------------------------------------------------------------------------------------

historical_contract_628hc7a11 = (
    FD002_TEST_PRESCORING_CONTRACT_628HC5A5
)

historical_identity_exact_628hc7a11 = bool(
    recomputed_fingerprints_628hc7a11["prediction_sha256"]
        == historical_contract_628hc7a11["prediction_sha256"]
    and recomputed_fingerprints_628hc7a11["risk_sha256"]
        == historical_contract_628hc7a11["risk_sha256"]
    and recomputed_fingerprints_628hc7a11["accept_mask_sha256"]
        == historical_contract_628hc7a11["accept_mask_sha256"]
    and recomputed_fingerprints_628hc7a11["abstain_mask_sha256"]
        == historical_contract_628hc7a11["abstain_mask_sha256"]
)

# --------------------------------------------------------------------------------------
# 7) Freeze-contract semantic audit
# --------------------------------------------------------------------------------------

contract_semantics_exact_628hc7a11 = bool(
    contract_628hc7a11.get("step") == "6.28H-C7-A10"
    and contract_628hc7a11.get("dataset") == "FD002"
    and contract_628hc7a11.get("freeze_path")
        == "verified_historical_prelabel_execution_reuse"
    and contract_628hc7a11.get("official_test_labels_admitted") is False
    and contract_628hc7a11.get("test_scored") is False
    and contract_628hc7a11.get("model_or_mc_rerun_performed") is False
    and contract_628hc7a11.get("reliability_architecture")
        == "uncertainty_only"
    and contract_628hc7a11.get("operating_policy")
        == "uncertainty_only_fixed_threshold"
    and contract_628hc7a11.get("comparator") == "<="
    and float(
        contract_628hc7a11.get("frozen_risk_threshold")
    ) == 0.8003345892330226
    and float(
        contract_628hc7a11.get("nominal_target_coverage")
    ) == 0.80
    and int(
        contract_628hc7a11.get("n_test_endpoints")
    ) == 259
    and int(
        contract_628hc7a11.get("mc_passes")
    ) == 50
    and int(
        contract_628hc7a11.get("mc_std_ddof")
    ) == 0
    and int(
        contract_628hc7a11.get("mc_seed")
    ) == 42
    and int(
        contract_628hc7a11.get("mc_batch_size")
    ) == 512
    and int(
        contract_628hc7a11.get("accepted_count")
    ) == 211
    and int(
        contract_628hc7a11.get("abstained_count")
    ) == 48
    and contract_628hc7a11.get(
        "resident_model_exact_historical_state_proven"
    ) is False
)

# --------------------------------------------------------------------------------------
# 8) Final C7-A11 audit
# --------------------------------------------------------------------------------------

audit_rows_628hc7a11 = [
    (
        "all_C7A10_fingerprints_recompute_exactly",
        all_fingerprints_exact_628hc7a11,
    ),
    (
        "C7A10_freeze_contract_SHA256_exact",
        contract_hash_exact_628hc7a11,
    ),
    (
        "frozen_numpy_arrays_are_read_only",
        arrays_read_only_628hc7a11,
    ),
    (
        "endpoint_metadata_exact_C7A7_locked_source",
        endpoint_metadata_exact_source_628hc7a11,
    ),
    (
        "MC_mean_exactly_rederived_from_frozen_matrix",
        mc_mean_exact_628hc7a11,
    ),
    (
        "MC_STD_exactly_rederived_ddof0",
        mc_std_exact_628hc7a11,
    ),
    (
        "risk_threshold_decision_rule_exact",
        decision_rule_exact_628hc7a11,
    ),
    (
        "accepted_abstained_counts_exact_211_48",
        counts_exact_628hc7a11,
    ),
    (
        "frozen_policy_threshold_comparator_exact",
        policy_exact_628hc7a11,
    ),
    (
        "shape_and_population_contract_exact",
        shape_contract_exact_628hc7a11,
    ),
    (
        "all_numeric_prelabel_artifacts_finite",
        finite_contract_exact_628hc7a11,
    ),
    (
        "historical_C5_prediction_risk_masks_identity_exact",
        historical_identity_exact_628hc7a11,
    ),
    (
        "freeze_contract_semantics_exact_and_prelabel",
        contract_semantics_exact_628hc7a11,
    ),
]

FD002_C7A11_AUDIT_628HC7A11 = pd.DataFrame(
    audit_rows_628hc7a11,
    columns=["check", "passed"],
)

all_checks_passed_628hc7a11 = bool(
    FD002_C7A11_AUDIT_628HC7A11[
        "passed"
    ].all()
)

if not all_checks_passed_628hc7a11:
    raise RuntimeError(
        "C7-A11 fingerprint/hash audit failed. "
        "Official FD002 RUL remains FORBIDDEN."
    )

# Authorization only — this does NOT read or admit official labels.
FD002_C7A11_OFFICIAL_RUL_ADMISSION_AUTHORIZED_628HC7A11 = True

FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11 = dict(
    recomputed_fingerprints_628hc7a11
)

FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11 = (
    recomputed_contract_sha_628hc7a11
)

# --------------------------------------------------------------------------------------
# 9) Output
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A11 — FD002 PRE-LABEL "
    "FINGERPRINT / HASH AUDIT"
)
print("=" * 128)

print("No official RUL/label object read.")
print("No TEST scoring performed.")
print("No model forward pass performed.")
print("No MC-dropout execution performed.")
print("No DEV retuning performed.")
print("No frozen policy/threshold modification performed.")
print("No files written.")

print("\n--- INDEPENDENT FINGERPRINT AUDIT ---")
print(
    FD002_C7A11_FINGERPRINT_AUDIT_628HC7A11.to_string(
        index=False
    )
)

print("\n--- C7-A11 AUDIT ---")
print(
    FD002_C7A11_AUDIT_628HC7A11.to_string(
        index=False
    )
)

print(
    "\nStored C7-A10 contract SHA-256     :",
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10,
)
print(
    "Recomputed contract SHA-256       :",
    recomputed_contract_sha_628hc7a11,
)
print(
    "Official RUL admission authorized :",
    FD002_C7A11_OFFICIAL_RUL_ADMISSION_AUTHORIZED_628HC7A11,
)

print(
    "\nPASS — C7-A11 complete. The current FD002 pre-label execution "
    "freeze is independently fingerprint-verified and unchanged."
)

print(
    "Next stage: STEP 6.28H-C7-A12 — one-time official FD002 RUL "
    "admission only. No scoring or policy changes in the admission step."
)

In [ ]:
# STEP 6.28H-C7-A12
# FD002 one-time official RUL admission ONLY
# NO TEST SCORING / NO ERROR COMPUTATION / NO POLICY OR THRESHOLD CHANGES

import hashlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions: C7-A11 must have independently verified the pre-label freeze
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a11", False)):
    raise RuntimeError(
        "C7-A11 must pass before official FD002 RUL admission."
    )

if not bool(
    globals().get(
        "FD002_C7A11_OFFICIAL_RUL_ADMISSION_AUTHORIZED_628HC7A11",
        False,
    )
):
    raise RuntimeError(
        "C7-A11 has not authorized official FD002 RUL admission."
    )

# Current-chain one-time guard.
if bool(
    globals().get(
        "FD002_C7A12_RUL_READ_ATTEMPTED_628HC7A12",
        False,
    )
):
    raise RuntimeError(
        "C7-A12 official RUL read has already been attempted in this "
        "current chain. Do not read RUL_FD002.txt again."
    )

required_628hc7a12 = [
    "FD002_C7A10_FROZEN_PREDICTION_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MEAN_628HC7A10",
    "FD002_C7A10_FROZEN_MC_STD_628HC7A10",
    "FD002_C7A10_FROZEN_RISK_628HC7A10",
    "FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10",
    "FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10",
    "FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11",
    "FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11",
]

missing_628hc7a12 = [
    name
    for name in required_628hc7a12
    if name not in globals()
]

if missing_628hc7a12:
    raise RuntimeError(
        "Missing required C7-A12 pre-label object(s): "
        + ", ".join(missing_628hc7a12)
    )

# --------------------------------------------------------------------------------------
# 2) LAST pre-admission integrity check
#    This section runs BEFORE official RUL bytes are opened.
# --------------------------------------------------------------------------------------

def _raw_array_sha256_628hc7a12(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype pre-label array refused during C7-A12."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _dataframe_sha256_628hc7a12(df):
    if not isinstance(df, pd.DataFrame):
        raise RuntimeError(
            "Frozen endpoint metadata is not a pandas DataFrame."
        )

    payload = df.to_csv(
        index=True,
        lineterminator="\n",
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


prelabel_current_628hc7a12 = {
    "prediction_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_PREDICTION_628HC7A10
        ),
    "mc_matrix_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10
        ),
    "mc_mean_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_MC_MEAN_628HC7A10
        ),
    "mc_std_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_MC_STD_628HC7A10
        ),
    "risk_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_RISK_628HC7A10
        ),
    "accept_mask_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
        ),
    "abstain_mask_sha256":
        _raw_array_sha256_628hc7a12(
            FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
        ),
    "endpoint_metadata_sha256":
        _dataframe_sha256_628hc7a12(
            FD002_C7A10_FROZEN_ENDPOINT_METADATA_628HC7A10
        ),
}

verified_prelabel_628hc7a12 = dict(
    FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11
)

prelabel_integrity_exact_628hc7a12 = bool(
    prelabel_current_628hc7a12
    ==
    verified_prelabel_628hc7a12
)

if not prelabel_integrity_exact_628hc7a12:
    raise RuntimeError(
        "Pre-label fingerprints changed after C7-A11. "
        "Official RUL admission is blocked."
    )

freeze_contract_payload_628hc7a12 = json.dumps(
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_628HC7A10,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

freeze_contract_sha_628hc7a12 = hashlib.sha256(
    freeze_contract_payload_628hc7a12
).hexdigest()

if (
    freeze_contract_sha_628hc7a12
    !=
    FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11
):
    raise RuntimeError(
        "Pre-label freeze-contract hash changed after C7-A11. "
        "Official RUL admission is blocked."
    )

# --------------------------------------------------------------------------------------
# 3) Exact sanctioned official source
# --------------------------------------------------------------------------------------

FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12 = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\RUL_FD002.txt"
)

if not FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12.is_file():
    raise FileNotFoundError(
        "Official FD002 RUL file not found at the locked path:\n"
        + str(FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12)
    )

# Set the one-time guard immediately before opening official label bytes.
FD002_C7A12_RUL_READ_ATTEMPTED_628HC7A12 = True

# --------------------------------------------------------------------------------------
# 4) ONE-TIME official RUL read
#    The file is opened once. Parsing occurs from the captured bytes.
# --------------------------------------------------------------------------------------

with FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12.open("rb") as f_628hc7a12:
    rul_file_bytes_628hc7a12 = f_628hc7a12.read()

if len(rul_file_bytes_628hc7a12) == 0:
    raise RuntimeError(
        "Official FD002 RUL file is empty."
    )

FD002_C7A12_OFFICIAL_RUL_FILE_SHA256_628HC7A12 = hashlib.sha256(
    rul_file_bytes_628hc7a12
).hexdigest()

rul_loaded_628hc7a12 = np.loadtxt(
    io.BytesIO(rul_file_bytes_628hc7a12),
    dtype=np.float64,
)

# NASA FD002 contract: one official remaining-cycles value per TEST engine.
if rul_loaded_628hc7a12.ndim != 1:
    raise RuntimeError(
        "Official FD002 RUL file is not a single-column vector."
    )

if rul_loaded_628hc7a12.shape != (259,):
    raise RuntimeError(
        "Official FD002 RUL vector must contain exactly 259 values; "
        f"found shape {rul_loaded_628hc7a12.shape}."
    )

if not np.isfinite(rul_loaded_628hc7a12).all():
    raise RuntimeError(
        "Official FD002 RUL vector contains non-finite values."
    )

if (rul_loaded_628hc7a12 < 0.0).any():
    raise RuntimeError(
        "Official FD002 RUL vector contains negative remaining-life values."
    )

# Freeze the admitted official vector in the current chain.
FD002_C7A12_OFFICIAL_RUL_628HC7A12 = np.array(
    rul_loaded_628hc7a12,
    dtype=np.float64,
    copy=True,
)

FD002_C7A12_OFFICIAL_RUL_628HC7A12.setflags(
    write=False
)

FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12 = (
    _raw_array_sha256_628hc7a12(
        FD002_C7A12_OFFICIAL_RUL_628HC7A12
    )
)

# --------------------------------------------------------------------------------------
# 5) Admission contract
#    IMPORTANT: no prediction/error/risk/decision computation occurs after label admission.
# --------------------------------------------------------------------------------------

FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12 = {
    "step": "6.28H-C7-A12",
    "dataset": "FD002",
    "action": "one_time_official_rul_admission_only",
    "source_path": str(
        FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12
    ),
    "source_file_sha256":
        FD002_C7A12_OFFICIAL_RUL_FILE_SHA256_628HC7A12,
    "official_rul_vector_sha256":
        FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12,
    "n_official_rul_values": 259,
    "official_rul_dtype": str(
        FD002_C7A12_OFFICIAL_RUL_628HC7A12.dtype
    ),
    "prelabel_freeze_contract_sha256":
        freeze_contract_sha_628hc7a12,
    "prelabel_integrity_verified_before_admission": True,
    "official_rul_admitted": True,
    "test_scored": False,
    "errors_computed": False,
    "policy_modified": False,
    "threshold_modified": False,
    "dev_revisited": False,
    "model_or_mc_rerun_performed": False,
}

rul_admission_payload_628hc7a12 = json.dumps(
    FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12 = (
    hashlib.sha256(
        rul_admission_payload_628hc7a12
    ).hexdigest()
)

FD002_C7A12_OFFICIAL_RUL_ADMITTED_628HC7A12 = True
FD002_C7A12_TEST_SCORED_628HC7A12 = False
all_checks_passed_628hc7a12 = True

# --------------------------------------------------------------------------------------
# 6) Output — structural/provenance information only; no label values displayed
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A12 — FD002 ONE-TIME OFFICIAL RUL ADMISSION"
)
print("=" * 128)

print(
    "Pre-label fingerprints unchanged before admission :",
    prelabel_integrity_exact_628hc7a12,
)
print(
    "Pre-label freeze-contract hash unchanged          :",
    True,
)
print(
    "Official RUL source                               :",
    FD002_C7A12_OFFICIAL_RUL_PATH_628HC7A12,
)
print(
    "Official RUL file opened in this step             :",
    True,
)
print(
    "Official RUL vector shape                         :",
    FD002_C7A12_OFFICIAL_RUL_628HC7A12.shape,
)
print(
    "Official RUL vector dtype                         :",
    FD002_C7A12_OFFICIAL_RUL_628HC7A12.dtype,
)
print(
    "Official RUL vector finite                        :",
    bool(
        np.isfinite(
            FD002_C7A12_OFFICIAL_RUL_628HC7A12
        ).all()
    ),
)
print(
    "Official RUL vector read-only                     :",
    bool(
        not FD002_C7A12_OFFICIAL_RUL_628HC7A12.flags.writeable
    ),
)
print(
    "Official RUL source-file SHA-256                   :",
    FD002_C7A12_OFFICIAL_RUL_FILE_SHA256_628HC7A12,
)
print(
    "Official RUL parsed-vector SHA-256                 :",
    FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12,
)
print(
    "RUL admission-contract SHA-256                    :",
    FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12,
)

print("\nTEST scoring performed                             :", False)
print("Error/MAE computation performed                   :", False)
print("Model/MC inference performed                      :", False)
print("DEV revisited                                     :", False)
print("Policy modified                                   :", False)
print("Threshold modified                                :", False)

print(
    "\nPASS — C7-A12 complete. The official FD002 RUL vector has been "
    "admitted exactly once into the current one-way evaluation chain."
)

print(
    "No TEST score or error has been computed. "
    "Next stage: STEP 6.28H-C7-A13 — reporting-only FD002 TEST scoring "
    "using the already-frozen prediction and accept/abstain decisions."
)

In [ ]:
# STEP 6.28H-C7-A13
# FD002 reporting-only TEST scoring
# Uses ONLY the frozen pre-label execution + admitted official RUL.
# NO RETUNING / NO POLICY CHANGES / NO MODEL OR MC INFERENCE

import hashlib
import json
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a12", False)):
    raise RuntimeError(
        "C7-A12 must pass before C7-A13 reporting-only TEST scoring."
    )

if not bool(
    globals().get(
        "FD002_C7A12_OFFICIAL_RUL_ADMITTED_628HC7A12",
        False,
    )
):
    raise RuntimeError(
        "Official FD002 RUL has not been admitted by C7-A12."
    )

if bool(
    globals().get(
        "FD002_C7A12_TEST_SCORED_628HC7A12",
        True,
    )
):
    raise RuntimeError(
        "C7-A12 state unexpectedly indicates TEST was already scored."
    )

if bool(
    globals().get(
        "FD002_C7A13_TEST_SCORED_628HC7A13",
        False,
    )
):
    raise RuntimeError(
        "C7-A13 has already completed in this current chain."
    )

required_628hc7a13 = [
    "FD002_C7A10_FROZEN_PREDICTION_628HC7A10",
    "FD002_C7A10_FROZEN_MC_STD_628HC7A10",
    "FD002_C7A10_FROZEN_RISK_628HC7A10",
    "FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_THRESHOLD_628HC7A10",
    "FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10",
    "FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11",
    "FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11",
    "FD002_C7A12_OFFICIAL_RUL_628HC7A12",
    "FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12",
    "FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12",
]

missing_628hc7a13 = [
    name
    for name in required_628hc7a13
    if name not in globals()
]

if missing_628hc7a13:
    raise RuntimeError(
        "Missing required C7-A13 object(s): "
        + ", ".join(missing_628hc7a13)
    )

# --------------------------------------------------------------------------------------
# 2) Integrity checks BEFORE computing any TEST error
# --------------------------------------------------------------------------------------

def _raw_array_sha256_628hc7a13(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during C7-A13."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


current_prelabel_fingerprints_628hc7a13 = {
    "prediction_sha256":
        _raw_array_sha256_628hc7a13(
            FD002_C7A10_FROZEN_PREDICTION_628HC7A10
        ),
    "mc_std_sha256":
        _raw_array_sha256_628hc7a13(
            FD002_C7A10_FROZEN_MC_STD_628HC7A10
        ),
    "risk_sha256":
        _raw_array_sha256_628hc7a13(
            FD002_C7A10_FROZEN_RISK_628HC7A10
        ),
    "accept_mask_sha256":
        _raw_array_sha256_628hc7a13(
            FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
        ),
    "abstain_mask_sha256":
        _raw_array_sha256_628hc7a13(
            FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
        ),
}

verified_prelabel_fingerprints_628hc7a13 = (
    FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11
)

for key_628hc7a13, sha_628hc7a13 in (
    current_prelabel_fingerprints_628hc7a13.items()
):
    if (
        verified_prelabel_fingerprints_628hc7a13.get(
            key_628hc7a13
        )
        != sha_628hc7a13
    ):
        raise RuntimeError(
            f"Pre-label artifact changed before scoring: {key_628hc7a13}"
        )

admission_payload_628hc7a13 = json.dumps(
    FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

admission_contract_sha_628hc7a13 = hashlib.sha256(
    admission_payload_628hc7a13
).hexdigest()

if (
    admission_contract_sha_628hc7a13
    !=
    FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12
):
    raise RuntimeError(
        "C7-A12 RUL-admission contract changed before scoring."
    )

if (
    FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12.get(
        "test_scored"
    )
    is not False
):
    raise RuntimeError(
        "C7-A12 admission contract does not certify test_scored=False."
    )

# --------------------------------------------------------------------------------------
# 3) Frozen evaluation inputs
# --------------------------------------------------------------------------------------

prediction_628hc7a13 = np.asarray(
    FD002_C7A10_FROZEN_PREDICTION_628HC7A10,
    dtype=np.float64,
)

official_rul_628hc7a13 = np.asarray(
    FD002_C7A12_OFFICIAL_RUL_628HC7A12,
    dtype=np.float64,
)

mc_std_628hc7a13 = np.asarray(
    FD002_C7A10_FROZEN_MC_STD_628HC7A10,
    dtype=np.float64,
)

risk_628hc7a13 = np.asarray(
    FD002_C7A10_FROZEN_RISK_628HC7A10,
    dtype=np.float64,
)

accept_628hc7a13 = np.asarray(
    FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10,
    dtype=bool,
)

abstain_628hc7a13 = np.asarray(
    FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10,
    dtype=bool,
)

if not (
    prediction_628hc7a13.shape
    == official_rul_628hc7a13.shape
    == risk_628hc7a13.shape
    == mc_std_628hc7a13.shape
    == accept_628hc7a13.shape
    == abstain_628hc7a13.shape
    == (259,)
):
    raise RuntimeError(
        "C7-A13 evaluation-array shape contract failed."
    )

if not np.array_equal(
    abstain_628hc7a13,
    ~accept_628hc7a13,
):
    raise RuntimeError(
        "Accept/abstain masks are no longer complementary."
    )

# --------------------------------------------------------------------------------------
# 4) REPORTING-ONLY TEST scoring
# --------------------------------------------------------------------------------------

signed_error_628hc7a13 = (
    prediction_628hc7a13
    -
    official_rul_628hc7a13
)

absolute_error_628hc7a13 = np.abs(
    signed_error_628hc7a13
)

full_mae_628hc7a13 = float(
    absolute_error_628hc7a13.mean()
)

accepted_mae_628hc7a13 = float(
    absolute_error_628hc7a13[
        accept_628hc7a13
    ].mean()
)

abstained_mae_628hc7a13 = float(
    absolute_error_628hc7a13[
        abstain_628hc7a13
    ].mean()
)

accepted_count_628hc7a13 = int(
    accept_628hc7a13.sum()
)

abstained_count_628hc7a13 = int(
    abstain_628hc7a13.sum()
)

realized_coverage_628hc7a13 = (
    accepted_count_628hc7a13
    /
    259.0
)

selective_mae_gain_628hc7a13 = float(
    full_mae_628hc7a13
    -
    accepted_mae_628hc7a13
)

relative_mae_reduction_pct_628hc7a13 = float(
    100.0
    * selective_mae_gain_628hc7a13
    / full_mae_628hc7a13
)

abstained_minus_accepted_mae_628hc7a13 = float(
    abstained_mae_628hc7a13
    -
    accepted_mae_628hc7a13
)

abstained_vs_accepted_error_increase_pct_628hc7a13 = float(
    100.0
    * abstained_minus_accepted_mae_628hc7a13
    / accepted_mae_628hc7a13
)

coverage_delta_vs_nominal_628hc7a13 = float(
    realized_coverage_628hc7a13
    -
    float(
        FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10
    )
)

# --------------------------------------------------------------------------------------
# 5) Freeze post-label reporting arrays
# --------------------------------------------------------------------------------------

FD002_C7A13_SIGNED_ERROR_628HC7A13 = np.array(
    signed_error_628hc7a13,
    dtype=np.float64,
    copy=True,
)

FD002_C7A13_ABSOLUTE_ERROR_628HC7A13 = np.array(
    absolute_error_628hc7a13,
    dtype=np.float64,
    copy=True,
)

for arr_628hc7a13 in [
    FD002_C7A13_SIGNED_ERROR_628HC7A13,
    FD002_C7A13_ABSOLUTE_ERROR_628HC7A13,
]:
    arr_628hc7a13.setflags(write=False)

FD002_C7A13_SCORING_TABLE_628HC7A13 = pd.DataFrame(
    {
        "unit_id": np.arange(
            1,
            260,
            dtype=np.int64,
        ),
        "prediction": prediction_628hc7a13,
        "official_rul": official_rul_628hc7a13,
        "signed_error": signed_error_628hc7a13,
        "absolute_error": absolute_error_628hc7a13,
        "mc_std": mc_std_628hc7a13,
        "reliability_risk": risk_628hc7a13,
        "accepted": accept_628hc7a13,
        "abstained": abstain_628hc7a13,
    }
)

FD002_C7A13_REPORTING_METRICS_628HC7A13 = {
    "dataset": "FD002",
    "n_test_endpoints": 259,
    "accepted_count": accepted_count_628hc7a13,
    "abstained_count": abstained_count_628hc7a13,
    "realized_coverage": realized_coverage_628hc7a13,
    "nominal_target_coverage":
        float(
            FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10
        ),
    "coverage_delta_vs_nominal":
        coverage_delta_vs_nominal_628hc7a13,
    "full_test_mae": full_mae_628hc7a13,
    "accepted_test_mae": accepted_mae_628hc7a13,
    "abstained_test_mae": abstained_mae_628hc7a13,
    "selective_mae_gain":
        selective_mae_gain_628hc7a13,
    "relative_mae_reduction_pct":
        relative_mae_reduction_pct_628hc7a13,
    "abstained_minus_accepted_mae":
        abstained_minus_accepted_mae_628hc7a13,
    "abstained_vs_accepted_error_increase_pct":
        abstained_vs_accepted_error_increase_pct_628hc7a13,
}

# --------------------------------------------------------------------------------------
# 6) Fingerprint the reporting-only evidence
# --------------------------------------------------------------------------------------

def _dataframe_sha256_628hc7a13(df):
    payload = df.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


FD002_C7A13_SIGNED_ERROR_SHA256_628HC7A13 = (
    _raw_array_sha256_628hc7a13(
        FD002_C7A13_SIGNED_ERROR_628HC7A13
    )
)

FD002_C7A13_ABSOLUTE_ERROR_SHA256_628HC7A13 = (
    _raw_array_sha256_628hc7a13(
        FD002_C7A13_ABSOLUTE_ERROR_628HC7A13
    )
)

FD002_C7A13_SCORING_TABLE_SHA256_628HC7A13 = (
    _dataframe_sha256_628hc7a13(
        FD002_C7A13_SCORING_TABLE_628HC7A13
    )
)

metrics_payload_628hc7a13 = json.dumps(
    FD002_C7A13_REPORTING_METRICS_628HC7A13,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD002_C7A13_REPORTING_METRICS_SHA256_628HC7A13 = (
    hashlib.sha256(
        metrics_payload_628hc7a13
    ).hexdigest()
)

FD002_C7A13_TEST_SCORED_628HC7A13 = True
FD002_C7A13_REPORTING_ONLY_628HC7A13 = True
FD002_C7A13_POLICY_MODIFIED_628HC7A13 = False
FD002_C7A13_THRESHOLD_MODIFIED_628HC7A13 = False
FD002_C7A13_DEV_REVISITED_628HC7A13 = False
FD002_C7A13_MODEL_OR_MC_RERUN_628HC7A13 = False
all_checks_passed_628hc7a13 = True

# --------------------------------------------------------------------------------------
# 7) Output
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A13 — FD002 REPORTING-ONLY TEST SCORING"
)
print("=" * 128)

print("Frozen pre-label fingerprints verified unchanged before scoring.")
print("C7-A12 RUL-admission contract verified unchanged.")
print("Historical post-label scoring objects were NOT inspected.")
print("No model forward pass performed.")
print("No MC-dropout execution performed.")
print("No DEV retuning performed.")
print("No policy or threshold modification performed.")
print("No files written.")

print("\n--- FD002 FINAL REPORTING-ONLY TEST METRICS ---")
print(
    "TEST endpoints                     :",
    259,
)
print(
    "Accepted / abstained               :",
    accepted_count_628hc7a13,
    "/",
    abstained_count_628hc7a13,
)
print(
    "Realized coverage                  :",
    realized_coverage_628hc7a13,
)
print(
    "Nominal target coverage            :",
    float(
        FD002_C7A10_NOMINAL_TARGET_COVERAGE_628HC7A10
    ),
)
print(
    "Coverage delta vs nominal          :",
    coverage_delta_vs_nominal_628hc7a13,
)

print(
    "\nFull TEST MAE                      :",
    full_mae_628hc7a13,
)
print(
    "Accepted TEST MAE                  :",
    accepted_mae_628hc7a13,
)
print(
    "Abstained TEST MAE                 :",
    abstained_mae_628hc7a13,
)
print(
    "Selective MAE gain (full-accepted) :",
    selective_mae_gain_628hc7a13,
)
print(
    "Relative MAE reduction (%)         :",
    relative_mae_reduction_pct_628hc7a13,
)
print(
    "Abstained - accepted MAE           :",
    abstained_minus_accepted_mae_628hc7a13,
)
print(
    "Abstained vs accepted increase (%) :",
    abstained_vs_accepted_error_increase_pct_628hc7a13,
)

print("\n--- REPORTING EVIDENCE FINGERPRINTS ---")
print(
    "Signed-error SHA-256               :",
    FD002_C7A13_SIGNED_ERROR_SHA256_628HC7A13,
)
print(
    "Absolute-error SHA-256             :",
    FD002_C7A13_ABSOLUTE_ERROR_SHA256_628HC7A13,
)
print(
    "Scoring-table SHA-256              :",
    FD002_C7A13_SCORING_TABLE_SHA256_628HC7A13,
)
print(
    "Reporting-metrics SHA-256          :",
    FD002_C7A13_REPORTING_METRICS_SHA256_628HC7A13,
)

print(
    "\nPASS — C7-A13 complete. FD002 TEST has been scored exactly once "
    "in the current frozen one-way chain for reporting only."
)

print(
    "No tuning or selection is permitted from these TEST results. "
    "Next stage: STEP 6.28H-C7-A14 — final FD002 evidence/fingerprint lock."
)

In [ ]:
# STEP 6.28H-C7-A14
# FD002 final reporting-evidence / fingerprint lock
# NO RETUNING / NO POLICY CHANGES / NO MODEL OR MC INFERENCE / NO NEW SCORING PATH

import hashlib
import json
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a13", False)):
    raise RuntimeError(
        "C7-A13 must pass before C7-A14 final FD002 evidence lock."
    )

if not bool(
    globals().get(
        "FD002_C7A13_TEST_SCORED_628HC7A13",
        False,
    )
):
    raise RuntimeError(
        "C7-A13 does not certify that FD002 TEST scoring completed."
    )

if not bool(
    globals().get(
        "FD002_C7A13_REPORTING_ONLY_628HC7A13",
        False,
    )
):
    raise RuntimeError(
        "C7-A13 is not marked reporting-only. Stop before final lock."
    )

if bool(
    globals().get(
        "FD002_C7A14_FINAL_CLOSED_628HC7A14",
        False,
    )
):
    raise RuntimeError(
        "C7-A14 final FD002 closure has already completed in this session."
    )

required_628hc7a14 = [
    # Frozen pre-label state
    "FD002_C7A10_FROZEN_PREDICTION_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10",
    "FD002_C7A10_FROZEN_MC_MEAN_628HC7A10",
    "FD002_C7A10_FROZEN_MC_STD_628HC7A10",
    "FD002_C7A10_FROZEN_RISK_628HC7A10",
    "FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10",
    "FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10",
    "FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10",
    "FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11",
    "FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11",

    # Official label admission
    "FD002_C7A12_OFFICIAL_RUL_628HC7A12",
    "FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12",
    "FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12",
    "FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12",

    # Reporting-only scoring evidence
    "FD002_C7A13_SIGNED_ERROR_628HC7A13",
    "FD002_C7A13_ABSOLUTE_ERROR_628HC7A13",
    "FD002_C7A13_SCORING_TABLE_628HC7A13",
    "FD002_C7A13_REPORTING_METRICS_628HC7A13",
    "FD002_C7A13_SIGNED_ERROR_SHA256_628HC7A13",
    "FD002_C7A13_ABSOLUTE_ERROR_SHA256_628HC7A13",
    "FD002_C7A13_SCORING_TABLE_SHA256_628HC7A13",
    "FD002_C7A13_REPORTING_METRICS_SHA256_628HC7A13",
]

missing_628hc7a14 = [
    name
    for name in required_628hc7a14
    if name not in globals()
]

if missing_628hc7a14:
    raise RuntimeError(
        "Missing required C7-A14 object(s): "
        + ", ".join(missing_628hc7a14)
    )

# --------------------------------------------------------------------------------------
# 2) Independent hashing helpers
# --------------------------------------------------------------------------------------

def _raw_array_sha256_628hc7a14(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during C7-A14."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _dataframe_sha256_628hc7a14(df):
    if not isinstance(df, pd.DataFrame):
        raise RuntimeError(
            "Expected pandas DataFrame during C7-A14."
        )

    payload = df.to_csv(
        index=False,
        lineterminator="\n",
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


# --------------------------------------------------------------------------------------
# 3) Re-verify PRE-LABEL state is still unchanged after scoring
# --------------------------------------------------------------------------------------

prelabel_recomputed_628hc7a14 = {
    "prediction_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_PREDICTION_628HC7A10
        ),
    "mc_matrix_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_MC_MATRIX_628HC7A10
        ),
    "mc_mean_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_MC_MEAN_628HC7A10
        ),
    "mc_std_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_MC_STD_628HC7A10
        ),
    "risk_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_RISK_628HC7A10
        ),
    "accept_mask_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10
        ),
    "abstain_mask_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10
        ),
}

verified_prelabel_628hc7a14 = dict(
    FD002_C7A11_VERIFIED_FINGERPRINTS_628HC7A11
)

for key_628hc7a14, sha_628hc7a14 in prelabel_recomputed_628hc7a14.items():

    if verified_prelabel_628hc7a14.get(
        key_628hc7a14
    ) != sha_628hc7a14:

        raise RuntimeError(
            f"Frozen pre-label artifact changed after scoring: "
            f"{key_628hc7a14}"
        )

prelabel_still_exact_628hc7a14 = True

if (
    FD002_C7A10_PRELABEL_FREEZE_CONTRACT_SHA256_628HC7A10
    !=
    FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11
):
    raise RuntimeError(
        "C7-A10/C7-A11 pre-label freeze-contract SHA mismatch."
    )

# --------------------------------------------------------------------------------------
# 4) Re-verify official RUL admission evidence
# --------------------------------------------------------------------------------------

official_rul_sha_628hc7a14 = _raw_array_sha256_628hc7a14(
    FD002_C7A12_OFFICIAL_RUL_628HC7A12
)

official_rul_exact_628hc7a14 = bool(
    official_rul_sha_628hc7a14
    ==
    FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12
)

if not official_rul_exact_628hc7a14:
    raise RuntimeError(
        "Official FD002 RUL vector changed after C7-A12 admission."
    )

admission_contract_payload_628hc7a14 = json.dumps(
    FD002_C7A12_RUL_ADMISSION_CONTRACT_628HC7A12,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

admission_contract_sha_628hc7a14 = hashlib.sha256(
    admission_contract_payload_628hc7a14
).hexdigest()

admission_contract_exact_628hc7a14 = bool(
    admission_contract_sha_628hc7a14
    ==
    FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12
)

if not admission_contract_exact_628hc7a14:
    raise RuntimeError(
        "C7-A12 RUL-admission contract changed after scoring."
    )

# --------------------------------------------------------------------------------------
# 5) Independently re-verify C7-A13 reporting evidence
# --------------------------------------------------------------------------------------

signed_error_sha_628hc7a14 = _raw_array_sha256_628hc7a14(
    FD002_C7A13_SIGNED_ERROR_628HC7A13
)

absolute_error_sha_628hc7a14 = _raw_array_sha256_628hc7a14(
    FD002_C7A13_ABSOLUTE_ERROR_628HC7A13
)

scoring_table_sha_628hc7a14 = _dataframe_sha256_628hc7a14(
    FD002_C7A13_SCORING_TABLE_628HC7A13
)

metrics_payload_628hc7a14 = json.dumps(
    FD002_C7A13_REPORTING_METRICS_628HC7A13,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

reporting_metrics_sha_628hc7a14 = hashlib.sha256(
    metrics_payload_628hc7a14
).hexdigest()

reporting_fingerprints_exact_628hc7a14 = bool(
    signed_error_sha_628hc7a14
        == FD002_C7A13_SIGNED_ERROR_SHA256_628HC7A13
    and absolute_error_sha_628hc7a14
        == FD002_C7A13_ABSOLUTE_ERROR_SHA256_628HC7A13
    and scoring_table_sha_628hc7a14
        == FD002_C7A13_SCORING_TABLE_SHA256_628HC7A13
    and reporting_metrics_sha_628hc7a14
        == FD002_C7A13_REPORTING_METRICS_SHA256_628HC7A13
)

if not reporting_fingerprints_exact_628hc7a14:
    raise RuntimeError(
        "C7-A13 reporting evidence changed before final lock."
    )

# --------------------------------------------------------------------------------------
# 6) Independent reporting-metric verification
#    This verifies C7-A13 only. It does NOT define or change a policy.
# --------------------------------------------------------------------------------------

prediction_628hc7a14 = np.asarray(
    FD002_C7A10_FROZEN_PREDICTION_628HC7A10,
    dtype=np.float64,
)

official_rul_628hc7a14 = np.asarray(
    FD002_C7A12_OFFICIAL_RUL_628HC7A12,
    dtype=np.float64,
)

accept_628hc7a14 = np.asarray(
    FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10,
    dtype=bool,
)

abstain_628hc7a14 = np.asarray(
    FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10,
    dtype=bool,
)

absolute_error_check_628hc7a14 = np.abs(
    prediction_628hc7a14
    -
    official_rul_628hc7a14
)

metrics_recomputed_628hc7a14 = {
    "dataset": "FD002",
    "n_test_endpoints": 259,
    "accepted_count": int(
        accept_628hc7a14.sum()
    ),
    "abstained_count": int(
        abstain_628hc7a14.sum()
    ),
    "realized_coverage": float(
        accept_628hc7a14.sum()
        /
        259.0
    ),
    "nominal_target_coverage": 0.80,
    "coverage_delta_vs_nominal": float(
        accept_628hc7a14.sum()
        /
        259.0
        -
        0.80
    ),
    "full_test_mae": float(
        absolute_error_check_628hc7a14.mean()
    ),
    "accepted_test_mae": float(
        absolute_error_check_628hc7a14[
            accept_628hc7a14
        ].mean()
    ),
    "abstained_test_mae": float(
        absolute_error_check_628hc7a14[
            abstain_628hc7a14
        ].mean()
    ),
}

metrics_recomputed_628hc7a14[
    "selective_mae_gain"
] = float(
    metrics_recomputed_628hc7a14[
        "full_test_mae"
    ]
    -
    metrics_recomputed_628hc7a14[
        "accepted_test_mae"
    ]
)

metrics_recomputed_628hc7a14[
    "relative_mae_reduction_pct"
] = float(
    100.0
    *
    metrics_recomputed_628hc7a14[
        "selective_mae_gain"
    ]
    /
    metrics_recomputed_628hc7a14[
        "full_test_mae"
    ]
)

metrics_recomputed_628hc7a14[
    "abstained_minus_accepted_mae"
] = float(
    metrics_recomputed_628hc7a14[
        "abstained_test_mae"
    ]
    -
    metrics_recomputed_628hc7a14[
        "accepted_test_mae"
    ]
)

metrics_recomputed_628hc7a14[
    "abstained_vs_accepted_error_increase_pct"
] = float(
    100.0
    *
    metrics_recomputed_628hc7a14[
        "abstained_minus_accepted_mae"
    ]
    /
    metrics_recomputed_628hc7a14[
        "accepted_test_mae"
    ]
)

metrics_exact_628hc7a14 = bool(
    metrics_recomputed_628hc7a14
    ==
    FD002_C7A13_REPORTING_METRICS_628HC7A13
)

if not metrics_exact_628hc7a14:
    raise RuntimeError(
        "Independent C7-A14 metric recomputation "
        "does not exactly match C7-A13."
    )

# --------------------------------------------------------------------------------------
# 7) Final reporting copies
# --------------------------------------------------------------------------------------

FD002_FINAL_TEST_PREDICTION_628HC7A14 = np.array(
    FD002_C7A10_FROZEN_PREDICTION_628HC7A10,
    dtype=np.float64,
    copy=True,
)

FD002_FINAL_TEST_RUL_628HC7A14 = np.array(
    FD002_C7A12_OFFICIAL_RUL_628HC7A12,
    dtype=np.float64,
    copy=True,
)

FD002_FINAL_TEST_RISK_628HC7A14 = np.array(
    FD002_C7A10_FROZEN_RISK_628HC7A10,
    dtype=np.float64,
    copy=True,
)

FD002_FINAL_ACCEPT_MASK_628HC7A14 = np.array(
    FD002_C7A10_FROZEN_ACCEPT_MASK_628HC7A10,
    dtype=bool,
    copy=True,
)

FD002_FINAL_ABSTAIN_MASK_628HC7A14 = np.array(
    FD002_C7A10_FROZEN_ABSTAIN_MASK_628HC7A10,
    dtype=bool,
    copy=True,
)

FD002_FINAL_ABSOLUTE_ERROR_628HC7A14 = np.array(
    FD002_C7A13_ABSOLUTE_ERROR_628HC7A13,
    dtype=np.float64,
    copy=True,
)

for arr_628hc7a14 in [
    FD002_FINAL_TEST_PREDICTION_628HC7A14,
    FD002_FINAL_TEST_RUL_628HC7A14,
    FD002_FINAL_TEST_RISK_628HC7A14,
    FD002_FINAL_ACCEPT_MASK_628HC7A14,
    FD002_FINAL_ABSTAIN_MASK_628HC7A14,
    FD002_FINAL_ABSOLUTE_ERROR_628HC7A14,
]:
    arr_628hc7a14.setflags(
        write=False
    )

FD002_FINAL_SCORING_TABLE_628HC7A14 = (
    FD002_C7A13_SCORING_TABLE_628HC7A13.copy(
        deep=True
    )
)

FD002_FINAL_REPORTING_METRICS_628HC7A14 = dict(
    FD002_C7A13_REPORTING_METRICS_628HC7A13
)

# --------------------------------------------------------------------------------------
# 8) Final evidence fingerprints
# --------------------------------------------------------------------------------------

FD002_FINAL_EVIDENCE_FINGERPRINTS_628HC7A14 = {
    "prelabel_freeze_contract_sha256":
        FD002_C7A11_VERIFIED_FREEZE_CONTRACT_SHA256_628HC7A11,

    "official_rul_vector_sha256":
        FD002_C7A12_OFFICIAL_RUL_VECTOR_SHA256_628HC7A12,

    "rul_admission_contract_sha256":
        FD002_C7A12_RUL_ADMISSION_CONTRACT_SHA256_628HC7A12,

    "final_prediction_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_FINAL_TEST_PREDICTION_628HC7A14
        ),

    "final_risk_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_FINAL_TEST_RISK_628HC7A14
        ),

    "final_accept_mask_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_FINAL_ACCEPT_MASK_628HC7A14
        ),

    "final_abstain_mask_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_FINAL_ABSTAIN_MASK_628HC7A14
        ),

    "final_absolute_error_sha256":
        _raw_array_sha256_628hc7a14(
            FD002_FINAL_ABSOLUTE_ERROR_628HC7A14
        ),

    "final_scoring_table_sha256":
        _dataframe_sha256_628hc7a14(
            FD002_FINAL_SCORING_TABLE_628HC7A14
        ),

    "final_reporting_metrics_sha256":
        reporting_metrics_sha_628hc7a14,
}

# --------------------------------------------------------------------------------------
# 9) Final closure contract
# --------------------------------------------------------------------------------------

FD002_FINAL_CLOSURE_CONTRACT_628HC7A14 = {
    "step": "6.28H-C7-A14",
    "dataset": "FD002",
    "status":
        "FINAL_CLOSED_REPORTING_ONLY",

    "research_role":
        "reliability-aware selective RUL evaluation under frozen policy",

    "reliability_architecture":
        "uncertainty_only",

    "operating_policy":
        "uncertainty_only_fixed_threshold",

    "frozen_threshold":
        0.8003345892330226,

    "comparator":
        "<=",

    "nominal_target_coverage":
        0.80,

    "n_test_endpoints":
        259,

    "accepted_count":
        int(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "accepted_count"
            ]
        ),

    "abstained_count":
        int(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "abstained_count"
            ]
        ),

    "realized_coverage":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "realized_coverage"
            ]
        ),

    "full_test_mae":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "full_test_mae"
            ]
        ),

    "accepted_test_mae":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "accepted_test_mae"
            ]
        ),

    "abstained_test_mae":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "abstained_test_mae"
            ]
        ),

    "selective_mae_gain":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "selective_mae_gain"
            ]
        ),

    "relative_mae_reduction_pct":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "relative_mae_reduction_pct"
            ]
        ),

    "abstained_minus_accepted_mae":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "abstained_minus_accepted_mae"
            ]
        ),

    "abstained_vs_accepted_error_increase_pct":
        float(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "abstained_vs_accepted_error_increase_pct"
            ]
        ),

    "policy_retuned_after_test":
        False,

    "threshold_changed_after_test":
        False,

    "model_retrained_after_test":
        False,

    "model_or_mc_rerun_for_final_score":
        False,

    "historical_postlabel_scoring_object_used":
        False,

    "resident_model_exact_historical_state_proven":
        False,

    "resident_model_limitation":
        "Current resident FD002-named weights are not claimed "
        "to be the exact original historical TEST-execution "
        "weights; final evaluation uses independently "
        "hash-verified historical pre-label outputs.",

    "evidence_fingerprints":
        dict(
            FD002_FINAL_EVIDENCE_FINGERPRINTS_628HC7A14
        ),
}

closure_payload_628hc7a14 = json.dumps(
    FD002_FINAL_CLOSURE_CONTRACT_628HC7A14,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD002_FINAL_CLOSURE_CONTRACT_SHA256_628HC7A14 = (
    hashlib.sha256(
        closure_payload_628hc7a14
    ).hexdigest()
)

# --------------------------------------------------------------------------------------
# 10) Final integrity audit
#     Performance direction is NOT a pass/fail criterion.
# --------------------------------------------------------------------------------------

FD002_C7A14_FINAL_AUDIT_628HC7A14 = pd.DataFrame(
    [
        (
            "prelabel_state_unchanged_after_scoring",
            prelabel_still_exact_628hc7a14,
        ),
        (
            "official_RUL_identity_unchanged",
            official_rul_exact_628hc7a14,
        ),
        (
            "RUL_admission_contract_unchanged",
            admission_contract_exact_628hc7a14,
        ),
        (
            "C7A13_reporting_fingerprints_exact",
            reporting_fingerprints_exact_628hc7a14,
        ),
        (
            "C7A13_metrics_independently_reproduced",
            metrics_exact_628hc7a14,
        ),
        (
            "accepted_count_211",
            int(
                accept_628hc7a14.sum()
            ) == 211,
        ),
        (
            "abstained_count_48",
            int(
                abstain_628hc7a14.sum()
            ) == 48,
        ),
        (
            "policy_retuning_after_TEST",
            False,
        ),
        (
            "threshold_change_after_TEST",
            False,
        ),
        (
            "model_retraining_after_TEST",
            False,
        ),
        (
            "model_or_MC_rerun_for_final_score",
            False,
        ),
        (
            "historical_postlabel_object_used_for_score",
            False,
        ),
    ],
    columns=[
        "check",
        "status",
    ],
)

required_true_checks_628hc7a14 = [
    "prelabel_state_unchanged_after_scoring",
    "official_RUL_identity_unchanged",
    "RUL_admission_contract_unchanged",
    "C7A13_reporting_fingerprints_exact",
    "C7A13_metrics_independently_reproduced",
    "accepted_count_211",
    "abstained_count_48",
]

required_false_checks_628hc7a14 = [
    "policy_retuning_after_TEST",
    "threshold_change_after_TEST",
    "model_retraining_after_TEST",
    "model_or_MC_rerun_for_final_score",
    "historical_postlabel_object_used_for_score",
]

audit_indexed_628hc7a14 = (
    FD002_C7A14_FINAL_AUDIT_628HC7A14
    .set_index("check")["status"]
)

true_checks_pass_628hc7a14 = bool(
    all(
        bool(
            audit_indexed_628hc7a14.loc[
                name_628hc7a14
            ]
        )
        for name_628hc7a14
        in required_true_checks_628hc7a14
    )
)

false_checks_pass_628hc7a14 = bool(
    all(
        not bool(
            audit_indexed_628hc7a14.loc[
                name_628hc7a14
            ]
        )
        for name_628hc7a14
        in required_false_checks_628hc7a14
    )
)

# Reporting observations only — never used to decide whether FD002 is accepted.
FD002_C7A14_REPORTING_OBSERVATIONS_628HC7A14 = {
    "accepted_MAE_lower_than_full_MAE":
        bool(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "accepted_test_mae"
            ]
            <
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "full_test_mae"
            ]
        ),

    "abstained_MAE_higher_than_accepted_MAE":
        bool(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "abstained_test_mae"
            ]
            >
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "accepted_test_mae"
            ]
        ),

    "selective_MAE_gain_positive":
        bool(
            FD002_FINAL_REPORTING_METRICS_628HC7A14[
                "selective_mae_gain"
            ]
            >
            0.0
        ),
}

all_checks_passed_628hc7a14 = bool(
    true_checks_pass_628hc7a14
    and
    false_checks_pass_628hc7a14
)

if not all_checks_passed_628hc7a14:
    raise RuntimeError(
        "C7-A14 final FD002 evidence audit failed."
    )

FD002_C7A14_FINAL_CLOSED_628HC7A14 = True

FD002_FINAL_STATUS_628HC7A14 = (
    "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED"
)

# --------------------------------------------------------------------------------------
# 11) Output
# --------------------------------------------------------------------------------------

print("=" * 128)
print(
    "STEP 6.28H-C7-A14 — FD002 FINAL EVIDENCE / FINGERPRINT LOCK"
)
print("=" * 128)

print("\n--- FINAL FD002 METRICS ---")

for key_628hc7a14, value_628hc7a14 in (
    FD002_FINAL_REPORTING_METRICS_628HC7A14.items()
):
    print(
        f"{key_628hc7a14:44s}:",
        value_628hc7a14,
    )

print("\n--- FINAL EVIDENCE FINGERPRINTS ---")

for key_628hc7a14, value_628hc7a14 in (
    FD002_FINAL_EVIDENCE_FINGERPRINTS_628HC7A14.items()
):
    print(
        f"{key_628hc7a14:44s}:",
        value_628hc7a14,
    )

print("\n--- FINAL C7-A14 INTEGRITY AUDIT ---")

print(
    FD002_C7A14_FINAL_AUDIT_628HC7A14.to_string(
        index=False
    )
)

print(
    "\n--- REPORTING OBSERVATIONS "
    "(NOT CLOSURE CRITERIA) ---"
)

for key_628hc7a14, value_628hc7a14 in (
    FD002_C7A14_REPORTING_OBSERVATIONS_628HC7A14.items()
):
    print(
        f"{key_628hc7a14:44s}:",
        value_628hc7a14,
    )

print(
    "\nFinal closure-contract SHA-256                  :",
    FD002_FINAL_CLOSURE_CONTRACT_SHA256_628HC7A14,
)

print(
    "FD002 final status                              :",
    FD002_FINAL_STATUS_628HC7A14,
)

print(
    "\nPASS — FD002 is now CLOSED. Its final TEST results are "
    "reporting-only and must never be used for retuning, "
    "architecture selection, threshold selection, or "
    "reliability-rule modification."
)

print(
    "\nNext project stage: FD003 development under the same "
    "governing research problem and leakage-safe methodology, "
    "with FD003-specific preprocessing, calibration, corruption "
    "analysis and policy freeze."
)

In [ ]:
# STEP 6.29A
# FD003 development entry: no-leakage boundary + source-path contract lock
# NO DATA CONTENT READ / NO TEST ACCESS / NO RUL ACCESS / NO TRAINING

import hashlib
import json
from pathlib import Path

# --------------------------------------------------------------------------------------
# 1) Preconditions: FD002 must be completely closed
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_628hc7a14", False)):
    raise RuntimeError(
        "FD002 C7-A14 final closure must pass before FD003 development begins."
    )

if not bool(
    globals().get(
        "FD002_C7A14_FINAL_CLOSED_628HC7A14",
        False,
    )
):
    raise RuntimeError(
        "FD002 is not marked final-closed. Stop before FD003."
    )

if str(
    globals().get(
        "FD002_FINAL_STATUS_628HC7A14",
        "",
    )
) != "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED":
    raise RuntimeError(
        "Unexpected FD002 final status. Stop before FD003."
    )

# --------------------------------------------------------------------------------------
# 2) Governing FD003 research boundary
# --------------------------------------------------------------------------------------

FD003_RESEARCH_PROBLEM_629A = (
    "Reliability-Aware Selective Remaining Useful Life Prediction "
    "under Sensor Degradation"
)

FD003_DATASET_629A = "FD003"

# Benchmark-level expectations from the governing Step-0 foundation.
# These are expectations to be verified later from the dataset itself where applicable.
FD003_EXPECTED_TRAIN_ENGINES_629A = 100
FD003_EXPECTED_TEST_ENGINES_629A = 100
FD003_EXPECTED_OPERATING_CONDITIONS_629A = 1
FD003_EXPECTED_FAULT_MODES_629A = 2

# Dataset-specific development must establish these independently.
# Nothing is inherited silently from FD001/FD002.
FD003_NORMALIZATION_LOCKED_629A = False
FD003_MODEL_LOCKED_629A = False
FD003_RELIABILITY_ARCHITECTURE_LOCKED_629A = False
FD003_OPERATING_POLICY_LOCKED_629A = False
FD003_THRESHOLD_LOCKED_629A = False

# Explicit no-leakage boundary at FD003 entry.
FD003_OFFICIAL_TEST_FEATURES_READ_629A = False
FD003_OFFICIAL_RUL_READ_629A = False
FD003_TEST_SCORED_629A = False
FD003_POST_TEST_RETUNING_PERMITTED_629A = False

# --------------------------------------------------------------------------------------
# 3) Exact expected source paths
#    Path existence is checked; file contents are NOT opened in this step.
# --------------------------------------------------------------------------------------

FD003_RAW_DATA_DIR_629A = Path(
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData"
)

FD003_TRAIN_PATH_629A = (
    FD003_RAW_DATA_DIR_629A
    / "train_FD003.txt"
)

FD003_TEST_PATH_629A = (
    FD003_RAW_DATA_DIR_629A
    / "test_FD003.txt"
)

FD003_RUL_PATH_629A = (
    FD003_RAW_DATA_DIR_629A
    / "RUL_FD003.txt"
)

FD003_SOURCE_PATHS_629A = {
    "train": str(FD003_TRAIN_PATH_629A),
    "test": str(FD003_TEST_PATH_629A),
    "rul": str(FD003_RUL_PATH_629A),
}

FD003_SOURCE_EXISTS_629A = {
    "train": bool(FD003_TRAIN_PATH_629A.is_file()),
    "test": bool(FD003_TEST_PATH_629A.is_file()),
    "rul": bool(FD003_RUL_PATH_629A.is_file()),
}

if not all(FD003_SOURCE_EXISTS_629A.values()):
    missing_paths_629a = [
        FD003_SOURCE_PATHS_629A[key]
        for key, present in FD003_SOURCE_EXISTS_629A.items()
        if not present
    ]

    raise FileNotFoundError(
        "Expected FD003 source file(s) are missing:\n"
        + "\n".join(missing_paths_629a)
    )

# --------------------------------------------------------------------------------------
# 4) Freeze the FD003 entry contract
# --------------------------------------------------------------------------------------

FD003_ENTRY_CONTRACT_629A = {
    "step": "6.29A",
    "dataset": "FD003",
    "research_problem": FD003_RESEARCH_PROBLEM_629A,

    "expected_train_engines": FD003_EXPECTED_TRAIN_ENGINES_629A,
    "expected_test_engines": FD003_EXPECTED_TEST_ENGINES_629A,
    "expected_operating_conditions": FD003_EXPECTED_OPERATING_CONDITIONS_629A,
    "expected_fault_modes": FD003_EXPECTED_FAULT_MODES_629A,

    "train_source_path": str(FD003_TRAIN_PATH_629A),
    "test_source_path": str(FD003_TEST_PATH_629A),
    "official_rul_source_path": str(FD003_RUL_PATH_629A),

    "source_paths_exist": dict(FD003_SOURCE_EXISTS_629A),

    "train_contents_read_in_this_step": False,
    "test_contents_read_in_this_step": False,
    "official_rul_contents_read_in_this_step": False,

    "test_scored": False,
    "normalization_locked": False,
    "model_locked": False,
    "reliability_architecture_locked": False,
    "operating_policy_locked": False,
    "threshold_locked": False,

    "FD001_threshold_transferred": False,
    "FD002_threshold_transferred": False,
    "FD001_normalizer_transferred": False,
    "FD002_normalizer_transferred": False,

    "post_test_retuning_permitted": False,

    "next_action":
        "read and verify FD003 TRAIN data contract only; "
        "keep TEST and official RUL unopened",
}

entry_payload_629a = json.dumps(
    FD003_ENTRY_CONTRACT_629A,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_ENTRY_CONTRACT_SHA256_629A = hashlib.sha256(
    entry_payload_629a
).hexdigest()

all_checks_passed_629a = True

# --------------------------------------------------------------------------------------
# 5) Output
# --------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STEP 6.29A — FD003 DEVELOPMENT ENTRY / "
    "NO-LEAKAGE + SOURCE-PATH CONTRACT LOCK"
)
print("=" * 120)

print("Governing research problem :", FD003_RESEARCH_PROBLEM_629A)
print("Current dataset            :", FD003_DATASET_629A)

print("\n--- BENCHMARK EXPECTATIONS TO VERIFY DURING FD003 ---")
print("Expected TRAIN engines     :", FD003_EXPECTED_TRAIN_ENGINES_629A)
print("Expected TEST engines      :", FD003_EXPECTED_TEST_ENGINES_629A)
print("Expected operating conds.  :", FD003_EXPECTED_OPERATING_CONDITIONS_629A)
print("Expected fault modes       :", FD003_EXPECTED_FAULT_MODES_629A)

print("\n--- LOCKED SOURCE PATHS ---")
print("TRAIN :", FD003_TRAIN_PATH_629A)
print("TEST  :", FD003_TEST_PATH_629A)
print("RUL   :", FD003_RUL_PATH_629A)

print("\n--- PATH EXISTENCE ---")
for key_629a, value_629a in FD003_SOURCE_EXISTS_629A.items():
    print(f"{key_629a:5s} :", value_629a)

print("\n--- FD003 ENTRY NO-LEAKAGE BOUNDARY ---")
print("TRAIN contents read        :", False)
print("TEST contents read         :", False)
print("Official RUL contents read :", False)
print("TEST scored                :", False)
print("Normalizer inherited       :", False)
print("Threshold inherited        :", False)
print("Policy inherited           :", False)
print("Post-TEST retuning allowed :", False)

print(
    "\nFD003 entry-contract SHA-256 :",
    FD003_ENTRY_CONTRACT_SHA256_629A,
)

print(
    "\nPASS — FD003 development is now formally opened with a clean "
    "dataset-specific boundary. No data contents were read in this step."
)

print(
    "Next: verify the FD003 TRAIN data contract only. "
    "TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29B
# FD003 TRAIN-only raw data contract verification
# TRAIN CONTENT READ ONLY / NO TEST ACCESS / NO RUL ACCESS / NO PREPROCESSING / NO SPLIT YET

import hashlib
import io
import json

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629a", False)):
    raise RuntimeError(
        "STEP 6.29A must pass before FD003 TRAIN data-contract verification."
    )

if bool(globals().get("FD003_OFFICIAL_TEST_FEATURES_READ_629A", True)):
    raise RuntimeError(
        "FD003 entry state unexpectedly indicates TEST features were already read."
    )

if bool(globals().get("FD003_OFFICIAL_RUL_READ_629A", True)):
    raise RuntimeError(
        "FD003 entry state unexpectedly indicates official RUL was already read."
    )

if "FD003_TRAIN_PATH_629A" not in globals():
    raise RuntimeError(
        "Locked FD003 TRAIN path from STEP 6.29A is missing."
    )

# --------------------------------------------------------------------------------------
# 2) Exact C-MAPSS raw column contract
# --------------------------------------------------------------------------------------

FD003_RAW_COLUMNS_629B = (
    ["unit_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

if len(FD003_RAW_COLUMNS_629B) != 26:
    raise RuntimeError(
        "Internal FD003 raw-column contract is not 26 columns."
    )

# --------------------------------------------------------------------------------------
# 3) Read FD003 TRAIN only
# --------------------------------------------------------------------------------------

with FD003_TRAIN_PATH_629A.open("rb") as f_629b:
    train_bytes_629b = f_629b.read()

if len(train_bytes_629b) == 0:
    raise RuntimeError(
        "FD003 TRAIN source is empty."
    )

FD003_TRAIN_SOURCE_FILE_SHA256_629B = hashlib.sha256(
    train_bytes_629b
).hexdigest()

train_raw_629b = pd.read_csv(
    io.BytesIO(train_bytes_629b),
    sep=r"\s+",
    header=None,
    engine="python",
)

if train_raw_629b.shape[1] != 26:
    raise RuntimeError(
        "FD003 TRAIN must parse to exactly 26 C-MAPSS columns; "
        f"found {train_raw_629b.shape[1]}."
    )

train_raw_629b.columns = FD003_RAW_COLUMNS_629B

# --------------------------------------------------------------------------------------
# 4) Strict raw structural checks
# --------------------------------------------------------------------------------------

numeric_matrix_629b = train_raw_629b.to_numpy(
    dtype=np.float64,
    copy=True,
)

all_finite_629b = bool(
    np.isfinite(numeric_matrix_629b).all()
)

if not all_finite_629b:
    raise RuntimeError(
        "FD003 TRAIN contains non-finite values."
    )

unit_values_629b = train_raw_629b["unit_id"].to_numpy(
    dtype=np.float64
)

cycle_values_629b = train_raw_629b["cycle"].to_numpy(
    dtype=np.float64
)

unit_integer_like_629b = bool(
    np.array_equal(
        unit_values_629b,
        np.round(unit_values_629b),
    )
)

cycle_integer_like_629b = bool(
    np.array_equal(
        cycle_values_629b,
        np.round(cycle_values_629b),
    )
)

if not unit_integer_like_629b:
    raise RuntimeError(
        "FD003 TRAIN unit_id contains non-integer values."
    )

if not cycle_integer_like_629b:
    raise RuntimeError(
        "FD003 TRAIN cycle contains non-integer values."
    )

train_raw_629b["unit_id"] = train_raw_629b["unit_id"].astype(
    np.int64
)

train_raw_629b["cycle"] = train_raw_629b["cycle"].astype(
    np.int64
)

unique_units_629b = np.sort(
    train_raw_629b["unit_id"].unique()
)

expected_units_629b = np.arange(
    1,
    FD003_EXPECTED_TRAIN_ENGINES_629A + 1,
    dtype=np.int64,
)

unit_id_contract_exact_629b = bool(
    np.array_equal(
        unique_units_629b,
        expected_units_629b,
    )
)

if not unit_id_contract_exact_629b:
    raise RuntimeError(
        "FD003 TRAIN unit IDs do not exactly match expected 1..100."
    )

duplicate_unit_cycle_count_629b = int(
    train_raw_629b.duplicated(
        subset=["unit_id", "cycle"]
    ).sum()
)

if duplicate_unit_cycle_count_629b != 0:
    raise RuntimeError(
        "FD003 TRAIN contains duplicate (unit_id, cycle) rows."
    )

sorted_index_629b = train_raw_629b.sort_values(
    ["unit_id", "cycle"],
    kind="mergesort",
).index.to_numpy()

file_order_exact_629b = bool(
    np.array_equal(
        sorted_index_629b,
        np.arange(len(train_raw_629b)),
    )
)

if not file_order_exact_629b:
    raise RuntimeError(
        "FD003 TRAIN is not ordered by unit_id then cycle."
    )

grouped_cycles_629b = train_raw_629b.groupby(
    "unit_id",
    sort=True,
)["cycle"]

cycle_min_by_unit_629b = grouped_cycles_629b.min()
cycle_max_by_unit_629b = grouped_cycles_629b.max()
cycle_count_by_unit_629b = grouped_cycles_629b.size()

all_engines_start_cycle_1_629b = bool(
    (cycle_min_by_unit_629b == 1).all()
)

contiguous_cycles_629b = bool(
    np.array_equal(
        cycle_max_by_unit_629b.to_numpy(),
        cycle_count_by_unit_629b.to_numpy(),
    )
)

if not all_engines_start_cycle_1_629b:
    raise RuntimeError(
        "At least one FD003 TRAIN engine does not start at cycle 1."
    )

if not contiguous_cycles_629b:
    raise RuntimeError(
        "At least one FD003 TRAIN engine has a cycle gap or duplicate."
    )

# --------------------------------------------------------------------------------------
# 5) Preserve authoritative parsed TRAIN object
# --------------------------------------------------------------------------------------

FD003_OFFICIAL_TRAIN_RAW_629B = train_raw_629b.copy(
    deep=True
)

FD003_TRAIN_CONTENTS_READ_629B = True
FD003_TEST_CONTENTS_READ_629B = False
FD003_RUL_CONTENTS_READ_629B = False

engine_lengths_629b = cycle_count_by_unit_629b.to_numpy(
    dtype=np.int64
)

op_ranges_629b = {
    col: {
        "min": float(
            FD003_OFFICIAL_TRAIN_RAW_629B[col].min()
        ),
        "max": float(
            FD003_OFFICIAL_TRAIN_RAW_629B[col].max()
        ),
    }
    for col in [
        "op_setting_1",
        "op_setting_2",
        "op_setting_3",
    ]
}

FD003_TRAIN_DATA_CONTRACT_629B = {
    "step": "6.29B",
    "dataset": "FD003",
    "source_path": str(FD003_TRAIN_PATH_629A),
    "source_file_sha256": FD003_TRAIN_SOURCE_FILE_SHA256_629B,
    "rows": int(FD003_OFFICIAL_TRAIN_RAW_629B.shape[0]),
    "columns": int(FD003_OFFICIAL_TRAIN_RAW_629B.shape[1]),
    "column_names": list(FD003_RAW_COLUMNS_629B),
    "n_train_engines": int(len(unique_units_629b)),
    "unit_ids_exact_1_to_100": unit_id_contract_exact_629b,
    "all_values_finite": all_finite_629b,
    "duplicate_unit_cycle_rows": duplicate_unit_cycle_count_629b,
    "file_order_unit_then_cycle": file_order_exact_629b,
    "all_engines_start_cycle_1": all_engines_start_cycle_1_629b,
    "all_engine_cycles_contiguous": contiguous_cycles_629b,
    "engine_length_min": int(engine_lengths_629b.min()),
    "engine_length_median": float(
        np.median(engine_lengths_629b)
    ),
    "engine_length_mean": float(
        engine_lengths_629b.mean()
    ),
    "engine_length_max": int(engine_lengths_629b.max()),
    "operational_setting_ranges": op_ranges_629b,
    "test_contents_read": False,
    "official_rul_contents_read": False,
    "preprocessing_performed": False,
    "fit_dev_split_performed": False,
}

contract_payload_629b = json.dumps(
    FD003_TRAIN_DATA_CONTRACT_629B,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_TRAIN_DATA_CONTRACT_SHA256_629B = hashlib.sha256(
    contract_payload_629b
).hexdigest()

all_checks_passed_629b = True

# --------------------------------------------------------------------------------------
# 6) Output
# --------------------------------------------------------------------------------------

print("=" * 124)
print(
    "STEP 6.29B — FD003 TRAIN-ONLY RAW DATA CONTRACT VERIFICATION"
)
print("=" * 124)

print("TRAIN contents read        :", True)
print("TEST contents read         :", False)
print("Official RUL contents read :", False)
print("Preprocessing performed    :", False)
print("FIT/DEV split performed    :", False)

print("\n--- RAW TRAIN CONTRACT ---")
print(
    "Source file SHA-256        :",
    FD003_TRAIN_SOURCE_FILE_SHA256_629B,
)
print(
    "Parsed shape               :",
    FD003_OFFICIAL_TRAIN_RAW_629B.shape,
)
print(
    "Column count               :",
    FD003_OFFICIAL_TRAIN_RAW_629B.shape[1],
)
print(
    "TRAIN engine count         :",
    len(unique_units_629b),
)
print(
    "Unit IDs exact 1..100      :",
    unit_id_contract_exact_629b,
)
print(
    "All numeric values finite  :",
    all_finite_629b,
)
print(
    "Duplicate unit-cycle rows  :",
    duplicate_unit_cycle_count_629b,
)
print(
    "Ordered unit then cycle    :",
    file_order_exact_629b,
)
print(
    "Every engine starts at 1   :",
    all_engines_start_cycle_1_629b,
)
print(
    "Every engine contiguous    :",
    contiguous_cycles_629b,
)

print("\n--- TRAIN ENGINE LENGTHS ---")
print(
    "Minimum cycles             :",
    int(engine_lengths_629b.min()),
)
print(
    "Median cycles              :",
    float(np.median(engine_lengths_629b)),
)
print(
    "Mean cycles                :",
    float(engine_lengths_629b.mean()),
)
print(
    "Maximum cycles             :",
    int(engine_lengths_629b.max()),
)

print("\n--- RAW OPERATIONAL-SETTING RANGES ---")
for col_629b, stats_629b in op_ranges_629b.items():
    print(
        f"{col_629b:16s}: "
        f"min={stats_629b['min']}, "
        f"max={stats_629b['max']}"
    )

print(
    "\nFD003 TRAIN data-contract SHA-256 :",
    FD003_TRAIN_DATA_CONTRACT_SHA256_629B,
)

print(
    "\nPASS — FD003 TRAIN raw structure is verified. "
    "TEST features and official RUL remain unopened."
)

print(
    "Next: establish the engine-aware FD003 FIT/DEV split contract "
    "before any preprocessing or model training."
)

In [ ]:
# STEP 6.29C
# FD003 engine-aware FIT/DEV split contract
# NO TEST ACCESS / NO RUL ACCESS / NO PREPROCESSING / NO TARGET CONSTRUCTION

import hashlib
import json

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629b", False)):
    raise RuntimeError(
        "STEP 6.29B must pass before the FD003 FIT/DEV split is established."
    )

if "FD003_OFFICIAL_TRAIN_RAW_629B" not in globals():
    raise RuntimeError(
        "Verified FD003 TRAIN object from STEP 6.29B is missing."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629B", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629B", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Freeze the split protocol BEFORE any DEV-outcome inspection
# --------------------------------------------------------------------------------------

FD003_SPLIT_SEED_629C = 62903
FD003_DEV_ENGINE_COUNT_629C = 20
FD003_FIT_ENGINE_COUNT_629C = 80
FD003_SPLIT_METHOD_629C = (
    "engine-level deterministic random permutation; "
    "first 20 shuffled engine IDs -> DEV; remaining 80 -> FIT"
)

train_raw_629c = FD003_OFFICIAL_TRAIN_RAW_629B

all_units_629c = np.sort(
    train_raw_629c["unit_id"].unique().astype(np.int64)
)

if not np.array_equal(
    all_units_629c,
    np.arange(1, 101, dtype=np.int64),
):
    raise RuntimeError(
        "FD003 TRAIN unit IDs changed since STEP 6.29B."
    )

rng_629c = np.random.default_rng(
    FD003_SPLIT_SEED_629C
)

shuffled_units_629c = rng_629c.permutation(
    all_units_629c
)

dev_units_629c = np.sort(
    shuffled_units_629c[
        :FD003_DEV_ENGINE_COUNT_629C
    ]
)

fit_units_629c = np.sort(
    shuffled_units_629c[
        FD003_DEV_ENGINE_COUNT_629C:
    ]
)

# --------------------------------------------------------------------------------------
# 3) Leakage-safety checks
# --------------------------------------------------------------------------------------

if len(fit_units_629c) != FD003_FIT_ENGINE_COUNT_629C:
    raise RuntimeError(
        "FD003 FIT engine count is not 80."
    )

if len(dev_units_629c) != FD003_DEV_ENGINE_COUNT_629C:
    raise RuntimeError(
        "FD003 DEV engine count is not 20."
    )

if len(np.intersect1d(fit_units_629c, dev_units_629c)) != 0:
    raise RuntimeError(
        "FD003 FIT/DEV engine overlap detected."
    )

if not np.array_equal(
    np.sort(
        np.concatenate(
            [fit_units_629c, dev_units_629c]
        )
    ),
    all_units_629c,
):
    raise RuntimeError(
        "FD003 FIT/DEV engine partition does not cover all 100 TRAIN engines."
    )

# --------------------------------------------------------------------------------------
# 4) Materialize TRAIN-derived FIT and DEV partitions
#    No preprocessing and no RUL target construction occurs here.
# --------------------------------------------------------------------------------------

fit_mask_629c = train_raw_629c["unit_id"].isin(
    fit_units_629c
)

dev_mask_629c = train_raw_629c["unit_id"].isin(
    dev_units_629c
)

FD003_FIT_RAW_629C = (
    train_raw_629c.loc[
        fit_mask_629c
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)

FD003_DEV_RAW_629C = (
    train_raw_629c.loc[
        dev_mask_629c
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)

if len(FD003_FIT_RAW_629C) + len(FD003_DEV_RAW_629C) != len(train_raw_629c):
    raise RuntimeError(
        "FD003 FIT/DEV row partition does not reconstruct full TRAIN."
    )

if set(FD003_FIT_RAW_629C["unit_id"].unique()) != set(fit_units_629c.tolist()):
    raise RuntimeError(
        "FD003 FIT DataFrame engine membership mismatch."
    )

if set(FD003_DEV_RAW_629C["unit_id"].unique()) != set(dev_units_629c.tolist()):
    raise RuntimeError(
        "FD003 DEV DataFrame engine membership mismatch."
    )

# --------------------------------------------------------------------------------------
# 5) Freeze engine-ID arrays
# --------------------------------------------------------------------------------------

FD003_FIT_ENGINE_IDS_629C = np.array(
    fit_units_629c,
    dtype=np.int64,
    copy=True,
)

FD003_DEV_ENGINE_IDS_629C = np.array(
    dev_units_629c,
    dtype=np.int64,
    copy=True,
)

FD003_FIT_ENGINE_IDS_629C.setflags(
    write=False
)

FD003_DEV_ENGINE_IDS_629C.setflags(
    write=False
)

# --------------------------------------------------------------------------------------
# 6) Split fingerprints and descriptive diagnostics
#    Diagnostics are reporting only; the seed is NOT changed based on them.
# --------------------------------------------------------------------------------------

def _raw_sha256_629c(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 split hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


fit_engine_lengths_629c = (
    FD003_FIT_RAW_629C
    .groupby("unit_id", sort=True)
    .size()
    .to_numpy(dtype=np.int64)
)

dev_engine_lengths_629c = (
    FD003_DEV_RAW_629C
    .groupby("unit_id", sort=True)
    .size()
    .to_numpy(dtype=np.int64)
)

FD003_FIT_ENGINE_IDS_SHA256_629C = _raw_sha256_629c(
    FD003_FIT_ENGINE_IDS_629C
)

FD003_DEV_ENGINE_IDS_SHA256_629C = _raw_sha256_629c(
    FD003_DEV_ENGINE_IDS_629C
)

FD003_SPLIT_CONTRACT_629C = {
    "step": "6.29C",
    "dataset": "FD003",
    "split_scope": "TRAIN engines only",
    "split_method": FD003_SPLIT_METHOD_629C,
    "split_seed": FD003_SPLIT_SEED_629C,
    "fit_engine_count": int(len(FD003_FIT_ENGINE_IDS_629C)),
    "dev_engine_count": int(len(FD003_DEV_ENGINE_IDS_629C)),
    "fit_row_count": int(len(FD003_FIT_RAW_629C)),
    "dev_row_count": int(len(FD003_DEV_RAW_629C)),
    "engine_overlap_count": int(
        len(
            np.intersect1d(
                FD003_FIT_ENGINE_IDS_629C,
                FD003_DEV_ENGINE_IDS_629C,
            )
        )
    ),
    "fit_engine_ids_sha256": FD003_FIT_ENGINE_IDS_SHA256_629C,
    "dev_engine_ids_sha256": FD003_DEV_ENGINE_IDS_SHA256_629C,
    "test_contents_read": False,
    "official_rul_contents_read": False,
    "preprocessing_performed": False,
    "rul_targets_constructed": False,
    "split_reseeded_after_diagnostics": False,
}

split_payload_629c = json.dumps(
    FD003_SPLIT_CONTRACT_629C,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_SPLIT_CONTRACT_SHA256_629C = hashlib.sha256(
    split_payload_629c
).hexdigest()

FD003_FIT_DEV_SPLIT_LOCKED_629C = True
FD003_TEST_CONTENTS_READ_629C = False
FD003_RUL_CONTENTS_READ_629C = False
all_checks_passed_629c = True

# --------------------------------------------------------------------------------------
# 7) Output
# --------------------------------------------------------------------------------------

print("=" * 124)
print(
    "STEP 6.29C — FD003 ENGINE-AWARE FIT/DEV SPLIT CONTRACT"
)
print("=" * 124)

print("Split scope                 : TRAIN engines only")
print("Split seed                  :", FD003_SPLIT_SEED_629C)
print("FIT engines                 :", len(FD003_FIT_ENGINE_IDS_629C))
print("DEV engines                 :", len(FD003_DEV_ENGINE_IDS_629C))
print("Engine overlap              :", 0)
print("TEST contents read          :", False)
print("Official RUL contents read  :", False)
print("Preprocessing performed     :", False)
print("RUL targets constructed     :", False)
print("Outcome-based reseeding     :", False)

print("\n--- FROZEN ENGINE IDS ---")
print(
    "FIT engine IDs :",
    FD003_FIT_ENGINE_IDS_629C.tolist(),
)
print(
    "DEV engine IDs :",
    FD003_DEV_ENGINE_IDS_629C.tolist(),
)

print("\n--- ROW COUNTS ---")
print("FIT rows                    :", len(FD003_FIT_RAW_629C))
print("DEV rows                    :", len(FD003_DEV_RAW_629C))
print(
    "FIT + DEV rows             :",
    len(FD003_FIT_RAW_629C) + len(FD003_DEV_RAW_629C),
)
print(
    "Original TRAIN rows        :",
    len(FD003_OFFICIAL_TRAIN_RAW_629B),
)

print("\n--- ENGINE-LENGTH DIAGNOSTICS (REPORTING ONLY) ---")
print(
    "FIT  min / median / mean / max :",
    int(fit_engine_lengths_629c.min()),
    "/",
    float(np.median(fit_engine_lengths_629c)),
    "/",
    float(fit_engine_lengths_629c.mean()),
    "/",
    int(fit_engine_lengths_629c.max()),
)
print(
    "DEV  min / median / mean / max :",
    int(dev_engine_lengths_629c.min()),
    "/",
    float(np.median(dev_engine_lengths_629c)),
    "/",
    float(dev_engine_lengths_629c.mean()),
    "/",
    int(dev_engine_lengths_629c.max()),
)

print("\n--- SPLIT FINGERPRINTS ---")
print(
    "FIT engine-ID SHA-256      :",
    FD003_FIT_ENGINE_IDS_SHA256_629C,
)
print(
    "DEV engine-ID SHA-256      :",
    FD003_DEV_ENGINE_IDS_SHA256_629C,
)
print(
    "Split-contract SHA-256     :",
    FD003_SPLIT_CONTRACT_SHA256_629C,
)

print(
    "\nPASS — FD003 FIT/DEV split is now engine-aware, deterministic, "
    "leakage-safe, and frozen before preprocessing or model selection."
)

print(
    "Next: define the FD003 TRAIN-derived RUL-label protocol and "
    "preprocessing contract using FIT statistics only. "
    "TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29D
# FD003 TRAIN-derived RUL-label contract + FIT-only raw feature variability audit
# NO TEST ACCESS / NO OFFICIAL RUL ACCESS / NO NORMALIZATION / NO FEATURE LOCK YET

import hashlib
import json

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629c", False)):
    raise RuntimeError(
        "STEP 6.29C must pass before FD003 label/preprocessing-contract work."
    )

required_629d = [
    "FD003_FIT_RAW_629C",
    "FD003_DEV_RAW_629C",
    "FD003_FIT_ENGINE_IDS_629C",
    "FD003_DEV_ENGINE_IDS_629C",
    "FD003_SPLIT_CONTRACT_SHA256_629C",
]

missing_629d = [
    name
    for name in required_629d
    if name not in globals()
]

if missing_629d:
    raise RuntimeError(
        "Missing required FD003 split object(s): "
        + ", ".join(missing_629d)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629C", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629C", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

fit_raw_629d = FD003_FIT_RAW_629C.copy(deep=True)
dev_raw_629d = FD003_DEV_RAW_629C.copy(deep=True)

# --------------------------------------------------------------------------------------
# 2) Standard TRAIN-derived per-cycle RUL labels
#    Governing Step-0 definition:
#    RUL = final observed TRAIN cycle for that engine - current cycle.
#
#    IMPORTANT:
#    - No arbitrary cap/clipping is introduced here.
#    - FIT and DEV are both subsets of original run-to-failure TRAIN trajectories.
# --------------------------------------------------------------------------------------

FD003_RUL_LABEL_PROTOCOL_629D = (
    "standard_uncapped_train_rul = engine_final_observed_cycle - current_cycle"
)

def _attach_standard_rul_629d(df):
    out = df.copy(deep=True)

    max_cycle_by_unit = out.groupby(
        "unit_id",
        sort=True,
    )["cycle"].transform("max")

    rul = (
        max_cycle_by_unit.to_numpy(dtype=np.int64)
        -
        out["cycle"].to_numpy(dtype=np.int64)
    )

    if (rul < 0).any():
        raise RuntimeError(
            "Negative TRAIN-derived RUL encountered."
        )

    out["rul"] = rul.astype(np.int64)

    return out


FD003_FIT_LABELED_RAW_629D = _attach_standard_rul_629d(
    fit_raw_629d
)

FD003_DEV_LABELED_RAW_629D = _attach_standard_rul_629d(
    dev_raw_629d
)

# Verify one zero-RUL endpoint per engine.
fit_zero_counts_629d = (
    FD003_FIT_LABELED_RAW_629D
    .groupby("unit_id", sort=True)["rul"]
    .apply(lambda s: int((s == 0).sum()))
)

dev_zero_counts_629d = (
    FD003_DEV_LABELED_RAW_629D
    .groupby("unit_id", sort=True)["rul"]
    .apply(lambda s: int((s == 0).sum()))
)

if not bool((fit_zero_counts_629d == 1).all()):
    raise RuntimeError(
        "FIT RUL label contract failed: expected exactly one zero-RUL endpoint per engine."
    )

if not bool((dev_zero_counts_629d == 1).all()):
    raise RuntimeError(
        "DEV RUL label contract failed: expected exactly one zero-RUL endpoint per engine."
    )

if int(FD003_FIT_LABELED_RAW_629D["rul"].min()) != 0:
    raise RuntimeError(
        "FIT RUL minimum is not zero."
    )

if int(FD003_DEV_LABELED_RAW_629D["rul"].min()) != 0:
    raise RuntimeError(
        "DEV RUL minimum is not zero."
    )

# --------------------------------------------------------------------------------------
# 3) FIT-only raw feature variability audit
#    No feature is selected or normalized in this step.
# --------------------------------------------------------------------------------------

FD003_RAW_FEATURE_CANDIDATES_629D = (
    [f"op_setting_{i}" for i in range(1, 4)]
    +
    [f"sensor_{i}" for i in range(1, 22)]
)

missing_feature_cols_629d = [
    col
    for col in FD003_RAW_FEATURE_CANDIDATES_629D
    if col not in FD003_FIT_LABELED_RAW_629D.columns
]

if missing_feature_cols_629d:
    raise RuntimeError(
        "Missing expected FD003 raw feature column(s): "
        + ", ".join(missing_feature_cols_629d)
    )

feature_rows_629d = []

for col_629d in FD003_RAW_FEATURE_CANDIDATES_629D:
    values_629d = FD003_FIT_LABELED_RAW_629D[
        col_629d
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(values_629d).all():
        raise RuntimeError(
            f"Non-finite FIT values found in {col_629d}."
        )

    min_629d = float(values_629d.min())
    max_629d = float(values_629d.max())
    range_629d = float(max_629d - min_629d)
    mean_629d = float(values_629d.mean())
    std_629d = float(values_629d.std(ddof=0))
    unique_629d = int(np.unique(values_629d).size)

    feature_rows_629d.append(
        {
            "feature": col_629d,
            "min": min_629d,
            "max": max_629d,
            "range": range_629d,
            "mean": mean_629d,
            "std_ddof0": std_629d,
            "unique_count": unique_629d,
            "exact_constant": bool(range_629d == 0.0),
            "zscore_possible": bool(std_629d > 0.0),
        }
    )

FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D = pd.DataFrame(
    feature_rows_629d
)

FD003_EXACT_CONSTANT_FEATURES_629D = (
    FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D.loc[
        FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D[
            "exact_constant"
        ],
        "feature",
    ]
    .tolist()
)

FD003_NONCONSTANT_FEATURES_629D = (
    FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D.loc[
        ~FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D[
            "exact_constant"
        ],
        "feature",
    ]
    .tolist()
)

# --------------------------------------------------------------------------------------
# 4) Fingerprint the derived label state
# --------------------------------------------------------------------------------------

def _raw_sha256_629d(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 label hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


FD003_FIT_RUL_SHA256_629D = _raw_sha256_629d(
    FD003_FIT_LABELED_RAW_629D["rul"].to_numpy(
        dtype=np.int64
    )
)

FD003_DEV_RUL_SHA256_629D = _raw_sha256_629d(
    FD003_DEV_LABELED_RAW_629D["rul"].to_numpy(
        dtype=np.int64
    )
)

# --------------------------------------------------------------------------------------
# 5) Contract
# --------------------------------------------------------------------------------------

FD003_LABEL_AND_VARIABILITY_CONTRACT_629D = {
    "step": "6.29D",
    "dataset": "FD003",
    "rul_label_protocol": FD003_RUL_LABEL_PROTOCOL_629D,
    "rul_cap_or_clipping_applied": False,
    "fit_rows": int(len(FD003_FIT_LABELED_RAW_629D)),
    "dev_rows": int(len(FD003_DEV_LABELED_RAW_629D)),
    "fit_engines": int(
        FD003_FIT_LABELED_RAW_629D["unit_id"].nunique()
    ),
    "dev_engines": int(
        FD003_DEV_LABELED_RAW_629D["unit_id"].nunique()
    ),
    "fit_rul_min": int(
        FD003_FIT_LABELED_RAW_629D["rul"].min()
    ),
    "fit_rul_max": int(
        FD003_FIT_LABELED_RAW_629D["rul"].max()
    ),
    "dev_rul_min": int(
        FD003_DEV_LABELED_RAW_629D["rul"].min()
    ),
    "dev_rul_max": int(
        FD003_DEV_LABELED_RAW_629D["rul"].max()
    ),
    "fit_rul_sha256": FD003_FIT_RUL_SHA256_629D,
    "dev_rul_sha256": FD003_DEV_RUL_SHA256_629D,
    "raw_feature_candidate_count": int(
        len(FD003_RAW_FEATURE_CANDIDATES_629D)
    ),
    "exact_constant_features":
        list(FD003_EXACT_CONSTANT_FEATURES_629D),
    "nonconstant_feature_count": int(
        len(FD003_NONCONSTANT_FEATURES_629D)
    ),
    "feature_selection_performed": False,
    "normalization_performed": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

contract_payload_629d = json.dumps(
    FD003_LABEL_AND_VARIABILITY_CONTRACT_629D,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_LABEL_AND_VARIABILITY_CONTRACT_SHA256_629D = (
    hashlib.sha256(
        contract_payload_629d
    ).hexdigest()
)

FD003_RUL_TARGETS_CONSTRUCTED_629D = True
FD003_FEATURE_SELECTION_LOCKED_629D = False
FD003_NORMALIZATION_LOCKED_629D = False
FD003_TEST_CONTENTS_READ_629D = False
FD003_RUL_CONTENTS_READ_629D = False
all_checks_passed_629d = True

# --------------------------------------------------------------------------------------
# 6) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29D — FD003 TRAIN-DERIVED RUL LABEL CONTRACT "
    "+ FIT-ONLY FEATURE VARIABILITY AUDIT"
)
print("=" * 126)

print("TEST contents read           :", False)
print("Official RUL contents read   :", False)
print("Feature selection performed  :", False)
print("Normalization performed      :", False)
print("RUL cap/clipping applied     :", False)

print("\n--- RUL LABEL CONTRACT ---")
print(
    "Protocol                     :",
    FD003_RUL_LABEL_PROTOCOL_629D,
)
print(
    "FIT  RUL min / max           :",
    int(FD003_FIT_LABELED_RAW_629D["rul"].min()),
    "/",
    int(FD003_FIT_LABELED_RAW_629D["rul"].max()),
)
print(
    "DEV  RUL min / max           :",
    int(FD003_DEV_LABELED_RAW_629D["rul"].min()),
    "/",
    int(FD003_DEV_LABELED_RAW_629D["rul"].max()),
)
print(
    "FIT zero-RUL endpoints       :",
    int((FD003_FIT_LABELED_RAW_629D["rul"] == 0).sum()),
)
print(
    "DEV zero-RUL endpoints       :",
    int((FD003_DEV_LABELED_RAW_629D["rul"] == 0).sum()),
)
print(
    "FIT RUL SHA-256              :",
    FD003_FIT_RUL_SHA256_629D,
)
print(
    "DEV RUL SHA-256              :",
    FD003_DEV_RUL_SHA256_629D,
)

print("\n--- FIT-ONLY RAW FEATURE VARIABILITY ---")
print(
    FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D.to_string(
        index=False
    )
)

print("\n--- VARIABILITY SUMMARY ---")
print(
    "Raw candidate features       :",
    len(FD003_RAW_FEATURE_CANDIDATES_629D),
)
print(
    "Exact-constant features      :",
    FD003_EXACT_CONSTANT_FEATURES_629D,
)
print(
    "Nonconstant features         :",
    len(FD003_NONCONSTANT_FEATURES_629D),
)

print(
    "\nContract SHA-256             :",
    FD003_LABEL_AND_VARIABILITY_CONTRACT_SHA256_629D,
)

print(
    "\nPASS — FD003 standard TRAIN-derived RUL labels are established "
    "without clipping, and FIT-only feature variability has been audited."
)

print(
    "Next: use this FIT-only evidence to lock the FD003 feature/"
    "normalization contract. TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29E
# FD003 feature-selection + FIT-only normalization contract lock
# NO TEST ACCESS / NO OFFICIAL RUL ACCESS / NO WINDOWING / NO MODEL TRAINING

import hashlib
import json

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629d", False)):
    raise RuntimeError(
        "STEP 6.29D must pass before FD003 feature/normalization locking."
    )

required_629e = [
    "FD003_FIT_LABELED_RAW_629D",
    "FD003_DEV_LABELED_RAW_629D",
    "FD003_RAW_FEATURE_CANDIDATES_629D",
    "FD003_EXACT_CONSTANT_FEATURES_629D",
    "FD003_NONCONSTANT_FEATURES_629D",
    "FD003_FIT_FEATURE_VARIABILITY_AUDIT_629D",
    "FD003_LABEL_AND_VARIABILITY_CONTRACT_SHA256_629D",
]

missing_629e = [
    name
    for name in required_629e
    if name not in globals()
]

if missing_629e:
    raise RuntimeError(
        "Missing required STEP 6.29D object(s): "
        + ", ".join(missing_629e)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629D", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629D", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Lock feature-selection rule
#
#    Rule:
#    - exclude FIT-exact-constant raw features only;
#    - retain every remaining nonconstant feature;
#    - do NOT force the historical FD001/FD002 17-feature signature onto FD003.
# --------------------------------------------------------------------------------------

FD003_FEATURE_SELECTION_RULE_629E = (
    "exclude FIT-exact-constant raw inputs only; "
    "retain all remaining nonconstant inputs"
)

expected_constants_629e = [
    "op_setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_16",
    "sensor_18",
    "sensor_19",
]

if list(FD003_EXACT_CONSTANT_FEATURES_629D) != expected_constants_629e:
    raise RuntimeError(
        "FD003 exact-constant feature set differs from the audited STEP 6.29D result."
    )

FD003_SELECTED_RAW_FEATURES_629E = list(
    FD003_NONCONSTANT_FEATURES_629D
)

if len(FD003_SELECTED_RAW_FEATURES_629E) != 18:
    raise RuntimeError(
        "Expected exactly 18 nonconstant FD003 FIT features."
    )

if set(FD003_SELECTED_RAW_FEATURES_629E).intersection(
    set(FD003_EXACT_CONSTANT_FEATURES_629D)
):
    raise RuntimeError(
        "Selected FD003 features unexpectedly include an exact-constant feature."
    )

# Preserve raw feature order from the original C-MAPSS column sequence.
expected_selected_order_629e = [
    col
    for col in FD003_RAW_FEATURE_CANDIDATES_629D
    if col not in set(FD003_EXACT_CONSTANT_FEATURES_629D)
]

if FD003_SELECTED_RAW_FEATURES_629E != expected_selected_order_629e:
    raise RuntimeError(
        "FD003 selected-feature order is inconsistent with the raw feature contract."
    )

# --------------------------------------------------------------------------------------
# 3) Compute normalization parameters using FIT ONLY
# --------------------------------------------------------------------------------------

fit_labeled_629e = FD003_FIT_LABELED_RAW_629D.copy(deep=True)
dev_labeled_629e = FD003_DEV_LABELED_RAW_629D.copy(deep=True)

fit_feature_matrix_629e = fit_labeled_629e[
    FD003_SELECTED_RAW_FEATURES_629E
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if not np.isfinite(fit_feature_matrix_629e).all():
    raise RuntimeError(
        "Non-finite FIT feature value encountered before normalization."
    )

FD003_FIT_FEATURE_MEANS_629E = fit_feature_matrix_629e.mean(
    axis=0
).astype(np.float64)

FD003_FIT_FEATURE_STDS_629E = fit_feature_matrix_629e.std(
    axis=0,
    ddof=0,
).astype(np.float64)

if not np.isfinite(FD003_FIT_FEATURE_MEANS_629E).all():
    raise RuntimeError(
        "Non-finite FIT normalization mean encountered."
    )

if not np.isfinite(FD003_FIT_FEATURE_STDS_629E).all():
    raise RuntimeError(
        "Non-finite FIT normalization standard deviation encountered."
    )

if not bool((FD003_FIT_FEATURE_STDS_629E > 0.0).all()):
    raise RuntimeError(
        "At least one selected FD003 feature has zero FIT standard deviation."
    )

FD003_FIT_FEATURE_MEANS_629E.setflags(write=False)
FD003_FIT_FEATURE_STDS_629E.setflags(write=False)

# --------------------------------------------------------------------------------------
# 4) Apply FIT-derived normalization to FIT and DEV
# --------------------------------------------------------------------------------------

FD003_NORMALIZED_FEATURE_NAMES_629E = [
    f"{col}_z"
    for col in FD003_SELECTED_RAW_FEATURES_629E
]

fit_z_629e = (
    fit_feature_matrix_629e
    -
    FD003_FIT_FEATURE_MEANS_629E
) / FD003_FIT_FEATURE_STDS_629E

dev_feature_matrix_629e = dev_labeled_629e[
    FD003_SELECTED_RAW_FEATURES_629E
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if not np.isfinite(dev_feature_matrix_629e).all():
    raise RuntimeError(
        "Non-finite DEV feature value encountered before normalization."
    )

dev_z_629e = (
    dev_feature_matrix_629e
    -
    FD003_FIT_FEATURE_MEANS_629E
) / FD003_FIT_FEATURE_STDS_629E

if not np.isfinite(fit_z_629e).all():
    raise RuntimeError(
        "Non-finite normalized FIT value encountered."
    )

if not np.isfinite(dev_z_629e).all():
    raise RuntimeError(
        "Non-finite normalized DEV value encountered."
    )

# --------------------------------------------------------------------------------------
# 5) Build normalized FIT / DEV tables
# --------------------------------------------------------------------------------------

FD003_FIT_NORMALIZED_629E = fit_labeled_629e.copy(deep=True)
FD003_DEV_NORMALIZED_629E = dev_labeled_629e.copy(deep=True)

for idx_629e, z_name_629e in enumerate(
    FD003_NORMALIZED_FEATURE_NAMES_629E
):
    FD003_FIT_NORMALIZED_629E[z_name_629e] = (
        fit_z_629e[:, idx_629e]
    )

    FD003_DEV_NORMALIZED_629E[z_name_629e] = (
        dev_z_629e[:, idx_629e]
    )

# --------------------------------------------------------------------------------------
# 6) Normalization verification
# --------------------------------------------------------------------------------------

fit_z_check_629e = FD003_FIT_NORMALIZED_629E[
    FD003_NORMALIZED_FEATURE_NAMES_629E
].to_numpy(
    dtype=np.float64
)

dev_z_check_629e = FD003_DEV_NORMALIZED_629E[
    FD003_NORMALIZED_FEATURE_NAMES_629E
].to_numpy(
    dtype=np.float64
)

fit_normalized_means_629e = fit_z_check_629e.mean(
    axis=0
)

fit_normalized_stds_629e = fit_z_check_629e.std(
    axis=0,
    ddof=0,
)

fit_means_near_zero_629e = bool(
    np.allclose(
        fit_normalized_means_629e,
        0.0,
        rtol=0.0,
        atol=1e-10,
    )
)

fit_stds_near_one_629e = bool(
    np.allclose(
        fit_normalized_stds_629e,
        1.0,
        rtol=0.0,
        atol=1e-10,
    )
)

if not fit_means_near_zero_629e:
    raise RuntimeError(
        "FIT normalized means are not numerically near zero."
    )

if not fit_stds_near_one_629e:
    raise RuntimeError(
        "FIT normalized standard deviations are not numerically near one."
    )

# DEV diagnostics are observation-only.
# They do not alter feature selection or scaler parameters.
dev_z_min_629e = np.min(
    dev_z_check_629e,
    axis=0,
)

dev_z_max_629e = np.max(
    dev_z_check_629e,
    axis=0,
)

# --------------------------------------------------------------------------------------
# 7) Fingerprints
# --------------------------------------------------------------------------------------

def _raw_sha256_629e(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 preprocessing hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


feature_list_payload_629e = json.dumps(
    FD003_SELECTED_RAW_FEATURES_629E,
    separators=(",", ":"),
).encode("utf-8")

normalized_feature_list_payload_629e = json.dumps(
    FD003_NORMALIZED_FEATURE_NAMES_629E,
    separators=(",", ":"),
).encode("utf-8")

FD003_SELECTED_FEATURE_LIST_SHA256_629E = hashlib.sha256(
    feature_list_payload_629e
).hexdigest()

FD003_NORMALIZED_FEATURE_LIST_SHA256_629E = hashlib.sha256(
    normalized_feature_list_payload_629e
).hexdigest()

FD003_FIT_MEANS_SHA256_629E = _raw_sha256_629e(
    FD003_FIT_FEATURE_MEANS_629E
)

FD003_FIT_STDS_SHA256_629E = _raw_sha256_629e(
    FD003_FIT_FEATURE_STDS_629E
)

FD003_FIT_NORMALIZED_MATRIX_SHA256_629E = _raw_sha256_629e(
    fit_z_check_629e
)

FD003_DEV_NORMALIZED_MATRIX_SHA256_629E = _raw_sha256_629e(
    dev_z_check_629e
)

# --------------------------------------------------------------------------------------
# 8) Freeze preprocessing contract
# --------------------------------------------------------------------------------------

FD003_PREPROCESSING_CONTRACT_629E = {
    "step": "6.29E",
    "dataset": "FD003",

    "feature_selection_rule":
        FD003_FEATURE_SELECTION_RULE_629E,

    "raw_candidate_count":
        int(len(FD003_RAW_FEATURE_CANDIDATES_629D)),

    "excluded_exact_constant_features":
        list(FD003_EXACT_CONSTANT_FEATURES_629D),

    "selected_feature_count":
        int(len(FD003_SELECTED_RAW_FEATURES_629E)),

    "selected_raw_features":
        list(FD003_SELECTED_RAW_FEATURES_629E),

    "normalized_feature_names":
        list(FD003_NORMALIZED_FEATURE_NAMES_629E),

    "normalization":
        "FIT-only z-score",

    "mean_source":
        "FD003 FIT rows only",

    "std_source":
        "FD003 FIT rows only",

    "std_ddof":
        0,

    "dev_used_to_fit_scaler":
        False,

    "test_used_to_fit_scaler":
        False,

    "official_rul_used":
        False,

    "feature_list_sha256":
        FD003_SELECTED_FEATURE_LIST_SHA256_629E,

    "normalized_feature_list_sha256":
        FD003_NORMALIZED_FEATURE_LIST_SHA256_629E,

    "fit_means_sha256":
        FD003_FIT_MEANS_SHA256_629E,

    "fit_stds_sha256":
        FD003_FIT_STDS_SHA256_629E,

    "fit_normalized_matrix_sha256":
        FD003_FIT_NORMALIZED_MATRIX_SHA256_629E,

    "dev_normalized_matrix_sha256":
        FD003_DEV_NORMALIZED_MATRIX_SHA256_629E,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629e = json.dumps(
    FD003_PREPROCESSING_CONTRACT_629E,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_PREPROCESSING_CONTRACT_SHA256_629E = hashlib.sha256(
    contract_payload_629e
).hexdigest()

FD003_FEATURE_SELECTION_LOCKED_629E = True
FD003_NORMALIZATION_LOCKED_629E = True
FD003_TEST_CONTENTS_READ_629E = False
FD003_RUL_CONTENTS_READ_629E = False
all_checks_passed_629e = True

# --------------------------------------------------------------------------------------
# 9) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29E — FD003 FEATURE + FIT-ONLY NORMALIZATION CONTRACT LOCK"
)
print("=" * 126)

print("TEST contents read             :", False)
print("Official RUL contents read     :", False)
print("DEV used to fit scaler         :", False)
print("Model training performed       :", False)
print("Window construction performed  :", False)

print("\n--- FEATURE CONTRACT ---")
print(
    "Selection rule                 :",
    FD003_FEATURE_SELECTION_RULE_629E,
)
print(
    "Raw candidate count            :",
    len(FD003_RAW_FEATURE_CANDIDATES_629D),
)
print(
    "Excluded exact constants       :",
    FD003_EXACT_CONSTANT_FEATURES_629D,
)
print(
    "Selected feature count         :",
    len(FD003_SELECTED_RAW_FEATURES_629E),
)
print(
    "Selected raw features          :",
    FD003_SELECTED_RAW_FEATURES_629E,
)
print(
    "Normalized model features      :",
    FD003_NORMALIZED_FEATURE_NAMES_629E,
)

print("\n--- FIT-ONLY NORMALIZATION CHECK ---")
print(
    "FIT normalized means ~ 0       :",
    fit_means_near_zero_629e,
)
print(
    "FIT normalized stds ~ 1        :",
    fit_stds_near_one_629e,
)
print(
    "FIT normalized finite          :",
    bool(np.isfinite(fit_z_check_629e).all()),
)
print(
    "DEV normalized finite          :",
    bool(np.isfinite(dev_z_check_629e).all()),
)

print("\n--- FIT SCALER PARAMETERS ---")

scaler_table_629e = pd.DataFrame(
    {
        "feature":
            FD003_SELECTED_RAW_FEATURES_629E,

        "normalized_feature":
            FD003_NORMALIZED_FEATURE_NAMES_629E,

        "fit_mean":
            FD003_FIT_FEATURE_MEANS_629E,

        "fit_std_ddof0":
            FD003_FIT_FEATURE_STDS_629E,

        "dev_z_min_reporting_only":
            dev_z_min_629e,

        "dev_z_max_reporting_only":
            dev_z_max_629e,
    }
)

print(
    scaler_table_629e.to_string(
        index=False
    )
)

print("\n--- PREPROCESSING FINGERPRINTS ---")
print(
    "Selected-feature SHA-256       :",
    FD003_SELECTED_FEATURE_LIST_SHA256_629E,
)
print(
    "Normalized-feature SHA-256     :",
    FD003_NORMALIZED_FEATURE_LIST_SHA256_629E,
)
print(
    "FIT means SHA-256              :",
    FD003_FIT_MEANS_SHA256_629E,
)
print(
    "FIT stds SHA-256               :",
    FD003_FIT_STDS_SHA256_629E,
)
print(
    "FIT normalized matrix SHA-256  :",
    FD003_FIT_NORMALIZED_MATRIX_SHA256_629E,
)
print(
    "DEV normalized matrix SHA-256  :",
    FD003_DEV_NORMALIZED_MATRIX_SHA256_629E,
)
print(
    "Preprocessing-contract SHA-256 :",
    FD003_PREPROCESSING_CONTRACT_SHA256_629E,
)

print(
    "\nPASS — FD003 feature selection and normalization are now locked "
    "using FIT information only."
)

print(
    "Next: establish the FD003 30-cycle FIT/DEV window contract "
    "and target alignment. TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29F
# FD003 30-cycle FIT/DEV window + endpoint-target alignment contract
# NO TEST ACCESS / NO OFFICIAL RUL ACCESS / NO MODEL TRAINING

import hashlib
import json

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629e", False)):
    raise RuntimeError(
        "STEP 6.29E must pass before FD003 window construction."
    )

required_629f = [
    "FD003_FIT_NORMALIZED_629E",
    "FD003_DEV_NORMALIZED_629E",
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_PREPROCESSING_CONTRACT_SHA256_629E",
    "FD003_FIT_ENGINE_IDS_629C",
    "FD003_DEV_ENGINE_IDS_629C",
]

missing_629f = [
    name
    for name in required_629f
    if name not in globals()
]

if missing_629f:
    raise RuntimeError(
        "Missing required STEP 6.29E/C object(s): "
        + ", ".join(missing_629f)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629E", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629E", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Window protocol
# --------------------------------------------------------------------------------------

FD003_WINDOW_LENGTH_629F = 30
FD003_WINDOW_STRIDE_629F = 1

FD003_WINDOW_TARGET_ALIGNMENT_629F = (
    "target = TRAIN-derived RUL at the final cycle of each 30-cycle window"
)

feature_cols_629f = list(
    FD003_NORMALIZED_FEATURE_NAMES_629E
)

if len(feature_cols_629f) != 18:
    raise RuntimeError(
        "FD003 normalized model feature count is not the locked value 18."
    )

# --------------------------------------------------------------------------------------
# 3) Deterministic engine-aware rolling-window builder
# --------------------------------------------------------------------------------------

def _build_windows_629f(
    df,
    feature_cols,
    window_length,
):
    if not isinstance(df, pd.DataFrame):
        raise RuntimeError(
            "Window source must be a pandas DataFrame."
        )

    required_cols = [
        "unit_id",
        "cycle",
        "rul",
    ] + list(feature_cols)

    missing_cols = [
        col
        for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise RuntimeError(
            "Window source missing column(s): "
            + ", ".join(missing_cols)
        )

    X_parts = []
    y_parts = []
    unit_parts = []
    endpoint_cycle_parts = []
    terminal_cycle_parts = []

    for unit_id, group in df.groupby(
        "unit_id",
        sort=True,
    ):
        group = group.sort_values(
            "cycle",
            kind="mergesort",
        )

        cycles = group[
            "cycle"
        ].to_numpy(
            dtype=np.int64
        )

        if len(cycles) < window_length:
            raise RuntimeError(
                f"Engine {unit_id} has fewer than "
                f"{window_length} cycles."
            )

        if not np.array_equal(
            cycles,
            np.arange(
                1,
                len(cycles) + 1,
                dtype=np.int64,
            ),
        ):
            raise RuntimeError(
                f"Engine {unit_id} cycle sequence is not "
                "contiguous from cycle 1."
            )

        features = group[
            feature_cols
        ].to_numpy(
            dtype=np.float32,
            copy=True,
        )

        rul = group[
            "rul"
        ].to_numpy(
            dtype=np.int64,
            copy=True,
        )

        terminal_cycle = int(
            cycles[-1]
        )

        expected_rul = (
            terminal_cycle
            -
            cycles
        )

        if not np.array_equal(
            rul,
            expected_rul,
        ):
            raise RuntimeError(
                f"Engine {unit_id} RUL labels do not match "
                "terminal_cycle - current_cycle."
            )

        n_windows = (
            len(group)
            -
            window_length
            +
            1
        )

        X_unit = np.empty(
            (
                n_windows,
                window_length,
                len(feature_cols),
            ),
            dtype=np.float32,
        )

        y_unit = np.empty(
            n_windows,
            dtype=np.float32,
        )

        endpoint_cycles_unit = np.empty(
            n_windows,
            dtype=np.int64,
        )

        for window_idx in range(
            n_windows
        ):
            start = window_idx

            stop = (
                start
                +
                window_length
            )

            endpoint_idx = (
                stop
                -
                1
            )

            X_unit[
                window_idx
            ] = features[
                start:stop
            ]

            y_unit[
                window_idx
            ] = float(
                rul[
                    endpoint_idx
                ]
            )

            endpoint_cycles_unit[
                window_idx
            ] = int(
                cycles[
                    endpoint_idx
                ]
            )

        X_parts.append(
            X_unit
        )

        y_parts.append(
            y_unit
        )

        unit_parts.append(
            np.full(
                n_windows,
                int(unit_id),
                dtype=np.int64,
            )
        )

        endpoint_cycle_parts.append(
            endpoint_cycles_unit
        )

        terminal_cycle_parts.append(
            np.full(
                n_windows,
                terminal_cycle,
                dtype=np.int64,
            )
        )

    X = np.concatenate(
        X_parts,
        axis=0,
    )

    y = np.concatenate(
        y_parts,
        axis=0,
    )

    unit_ids = np.concatenate(
        unit_parts,
        axis=0,
    )

    endpoint_cycles = np.concatenate(
        endpoint_cycle_parts,
        axis=0,
    )

    terminal_cycles = np.concatenate(
        terminal_cycle_parts,
        axis=0,
    )

    return (
        X,
        y,
        unit_ids,
        endpoint_cycles,
        terminal_cycles,
    )


(
    X_FIT_FD003_629F,
    y_FIT_FD003_629F,
    FIT_WINDOW_UNIT_IDS_FD003_629F,
    FIT_WINDOW_ENDPOINT_CYCLES_FD003_629F,
    FIT_WINDOW_TERMINAL_CYCLES_FD003_629F,
) = _build_windows_629f(
    FD003_FIT_NORMALIZED_629E,
    feature_cols_629f,
    FD003_WINDOW_LENGTH_629F,
)

(
    X_DEV_FD003_629F,
    y_DEV_FD003_629F,
    DEV_WINDOW_UNIT_IDS_FD003_629F,
    DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
    DEV_WINDOW_TERMINAL_CYCLES_FD003_629F,
) = _build_windows_629f(
    FD003_DEV_NORMALIZED_629E,
    feature_cols_629f,
    FD003_WINDOW_LENGTH_629F,
)

# --------------------------------------------------------------------------------------
# 4) Exact expected-count verification
# --------------------------------------------------------------------------------------

expected_fit_windows_629f = int(
    len(
        FD003_FIT_NORMALIZED_629E
    )
    -
    (
        FD003_WINDOW_LENGTH_629F
        -
        1
    )
    *
    len(
        FD003_FIT_ENGINE_IDS_629C
    )
)

expected_dev_windows_629f = int(
    len(
        FD003_DEV_NORMALIZED_629E
    )
    -
    (
        FD003_WINDOW_LENGTH_629F
        -
        1
    )
    *
    len(
        FD003_DEV_ENGINE_IDS_629C
    )
)

if X_FIT_FD003_629F.shape != (
    expected_fit_windows_629f,
    30,
    18,
):
    raise RuntimeError(
        "FD003 FIT window tensor shape mismatch."
    )

if X_DEV_FD003_629F.shape != (
    expected_dev_windows_629f,
    30,
    18,
):
    raise RuntimeError(
        "FD003 DEV window tensor shape mismatch."
    )

if y_FIT_FD003_629F.shape != (
    expected_fit_windows_629f,
):
    raise RuntimeError(
        "FD003 FIT target shape mismatch."
    )

if y_DEV_FD003_629F.shape != (
    expected_dev_windows_629f,
):
    raise RuntimeError(
        "FD003 DEV target shape mismatch."
    )

# --------------------------------------------------------------------------------------
# 5) Alignment / leakage-safety checks
# --------------------------------------------------------------------------------------

fit_target_alignment_exact_629f = bool(
    np.array_equal(
        y_FIT_FD003_629F.astype(
            np.int64
        ),
        (
            FIT_WINDOW_TERMINAL_CYCLES_FD003_629F
            -
            FIT_WINDOW_ENDPOINT_CYCLES_FD003_629F
        ),
    )
)

dev_target_alignment_exact_629f = bool(
    np.array_equal(
        y_DEV_FD003_629F.astype(
            np.int64
        ),
        (
            DEV_WINDOW_TERMINAL_CYCLES_FD003_629F
            -
            DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F
        ),
    )
)

if not fit_target_alignment_exact_629f:
    raise RuntimeError(
        "FD003 FIT window target alignment failed."
    )

if not dev_target_alignment_exact_629f:
    raise RuntimeError(
        "FD003 DEV window target alignment failed."
    )

fit_engine_membership_exact_629f = bool(
    set(
        np.unique(
            FIT_WINDOW_UNIT_IDS_FD003_629F
        ).tolist()
    )
    ==
    set(
        FD003_FIT_ENGINE_IDS_629C.tolist()
    )
)

dev_engine_membership_exact_629f = bool(
    set(
        np.unique(
            DEV_WINDOW_UNIT_IDS_FD003_629F
        ).tolist()
    )
    ==
    set(
        FD003_DEV_ENGINE_IDS_629C.tolist()
    )
)

if not fit_engine_membership_exact_629f:
    raise RuntimeError(
        "FD003 FIT window engine membership mismatch."
    )

if not dev_engine_membership_exact_629f:
    raise RuntimeError(
        "FD003 DEV window engine membership mismatch."
    )

if len(
    np.intersect1d(
        np.unique(
            FIT_WINDOW_UNIT_IDS_FD003_629F
        ),
        np.unique(
            DEV_WINDOW_UNIT_IDS_FD003_629F
        ),
    )
) != 0:
    raise RuntimeError(
        "FD003 FIT/DEV window engine leakage detected."
    )

if not np.isfinite(
    X_FIT_FD003_629F
).all():
    raise RuntimeError(
        "FD003 FIT windows contain non-finite values."
    )

if not np.isfinite(
    X_DEV_FD003_629F
).all():
    raise RuntimeError(
        "FD003 DEV windows contain non-finite values."
    )

if not np.isfinite(
    y_FIT_FD003_629F
).all():
    raise RuntimeError(
        "FD003 FIT targets contain non-finite values."
    )

if not np.isfinite(
    y_DEV_FD003_629F
).all():
    raise RuntimeError(
        "FD003 DEV targets contain non-finite values."
    )

# --------------------------------------------------------------------------------------
# 6) Freeze arrays read-only
# --------------------------------------------------------------------------------------

for arr_629f in [
    X_FIT_FD003_629F,
    y_FIT_FD003_629F,
    FIT_WINDOW_UNIT_IDS_FD003_629F,
    FIT_WINDOW_ENDPOINT_CYCLES_FD003_629F,
    FIT_WINDOW_TERMINAL_CYCLES_FD003_629F,
    X_DEV_FD003_629F,
    y_DEV_FD003_629F,
    DEV_WINDOW_UNIT_IDS_FD003_629F,
    DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
    DEV_WINDOW_TERMINAL_CYCLES_FD003_629F,
]:
    arr_629f.setflags(
        write=False
    )

# --------------------------------------------------------------------------------------
# 7) Fingerprints
# --------------------------------------------------------------------------------------

def _raw_sha256_629f(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 window hashing."
        )

    return hashlib.sha256(
        arr.tobytes(
            order="C"
        )
    ).hexdigest()


FD003_FIT_WINDOW_SHA256_629F = _raw_sha256_629f(
    X_FIT_FD003_629F
)

FD003_FIT_TARGET_SHA256_629F = _raw_sha256_629f(
    y_FIT_FD003_629F
)

FD003_DEV_WINDOW_SHA256_629F = _raw_sha256_629f(
    X_DEV_FD003_629F
)

FD003_DEV_TARGET_SHA256_629F = _raw_sha256_629f(
    y_DEV_FD003_629F
)

FD003_FIT_WINDOW_KEYS_SHA256_629F = _raw_sha256_629f(
    np.column_stack(
        [
            FIT_WINDOW_UNIT_IDS_FD003_629F,
            FIT_WINDOW_ENDPOINT_CYCLES_FD003_629F,
        ]
    ).astype(
        np.int64
    )
)

FD003_DEV_WINDOW_KEYS_SHA256_629F = _raw_sha256_629f(
    np.column_stack(
        [
            DEV_WINDOW_UNIT_IDS_FD003_629F,
            DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
        ]
    ).astype(
        np.int64
    )
)

# --------------------------------------------------------------------------------------
# 8) Window contract
# --------------------------------------------------------------------------------------

FD003_WINDOW_CONTRACT_629F = {
    "step": "6.29F",
    "dataset": "FD003",

    "window_length": 30,
    "window_stride": 1,

    "model_feature_count": 18,

    "model_features":
        list(
            FD003_NORMALIZED_FEATURE_NAMES_629E
        ),

    "target_alignment":
        FD003_WINDOW_TARGET_ALIGNMENT_629F,

    "target_protocol":
        "uncapped TRAIN-derived RUL from STEP 6.29D",

    "fit_engine_count":
        int(
            len(
                FD003_FIT_ENGINE_IDS_629C
            )
        ),

    "dev_engine_count":
        int(
            len(
                FD003_DEV_ENGINE_IDS_629C
            )
        ),

    "fit_window_count":
        int(
            len(
                X_FIT_FD003_629F
            )
        ),

    "dev_window_count":
        int(
            len(
                X_DEV_FD003_629F
            )
        ),

    "fit_expected_window_count":
        expected_fit_windows_629f,

    "dev_expected_window_count":
        expected_dev_windows_629f,

    "fit_target_alignment_exact":
        fit_target_alignment_exact_629f,

    "dev_target_alignment_exact":
        dev_target_alignment_exact_629f,

    "fit_dev_engine_overlap":
        0,

    "input_dtype":
        str(
            X_FIT_FD003_629F.dtype
        ),

    "target_dtype":
        str(
            y_FIT_FD003_629F.dtype
        ),

    "fit_window_sha256":
        FD003_FIT_WINDOW_SHA256_629F,

    "fit_target_sha256":
        FD003_FIT_TARGET_SHA256_629F,

    "dev_window_sha256":
        FD003_DEV_WINDOW_SHA256_629F,

    "dev_target_sha256":
        FD003_DEV_TARGET_SHA256_629F,

    "fit_window_keys_sha256":
        FD003_FIT_WINDOW_KEYS_SHA256_629F,

    "dev_window_keys_sha256":
        FD003_DEV_WINDOW_KEYS_SHA256_629F,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "model_training_performed":
        False,
}

contract_payload_629f = json.dumps(
    FD003_WINDOW_CONTRACT_629F,
    sort_keys=True,
    separators=(",", ":"),
).encode(
    "utf-8"
)

FD003_WINDOW_CONTRACT_SHA256_629F = hashlib.sha256(
    contract_payload_629f
).hexdigest()

FD003_WINDOW_CONTRACT_LOCKED_629F = True
FD003_TEST_CONTENTS_READ_629F = False
FD003_RUL_CONTENTS_READ_629F = False
all_checks_passed_629f = True

# --------------------------------------------------------------------------------------
# 9) Output
# --------------------------------------------------------------------------------------

print("=" * 126)

print(
    "STEP 6.29F — FD003 30-CYCLE FIT/DEV WINDOW "
    "+ ENDPOINT-TARGET ALIGNMENT CONTRACT"
)

print("=" * 126)

print(
    "TEST contents read            :",
    False,
)

print(
    "Official RUL contents read    :",
    False,
)

print(
    "Model training performed      :",
    False,
)

print("\n--- WINDOW PROTOCOL ---")

print(
    "Window length                 :",
    FD003_WINDOW_LENGTH_629F,
)

print(
    "Stride                        :",
    FD003_WINDOW_STRIDE_629F,
)

print(
    "Model feature count           :",
    len(
        feature_cols_629f
    ),
)

print(
    "Input dtype                   :",
    X_FIT_FD003_629F.dtype,
)

print(
    "Target dtype                  :",
    y_FIT_FD003_629F.dtype,
)

print(
    "Target alignment              :",
    FD003_WINDOW_TARGET_ALIGNMENT_629F,
)

print("\n--- FIT WINDOW CONTRACT ---")

print(
    "FIT tensor shape              :",
    X_FIT_FD003_629F.shape,
)

print(
    "FIT target shape              :",
    y_FIT_FD003_629F.shape,
)

print(
    "FIT expected windows          :",
    expected_fit_windows_629f,
)

print(
    "FIT target alignment exact    :",
    fit_target_alignment_exact_629f,
)

print(
    "FIT engine membership exact   :",
    fit_engine_membership_exact_629f,
)

print(
    "FIT target min / max          :",
    float(
        y_FIT_FD003_629F.min()
    ),
    "/",
    float(
        y_FIT_FD003_629F.max()
    ),
)

print("\n--- DEV WINDOW CONTRACT ---")

print(
    "DEV tensor shape              :",
    X_DEV_FD003_629F.shape,
)

print(
    "DEV target shape              :",
    y_DEV_FD003_629F.shape,
)

print(
    "DEV expected windows          :",
    expected_dev_windows_629f,
)

print(
    "DEV target alignment exact    :",
    dev_target_alignment_exact_629f,
)

print(
    "DEV engine membership exact   :",
    dev_engine_membership_exact_629f,
)

print(
    "DEV target min / max          :",
    float(
        y_DEV_FD003_629F.min()
    ),
    "/",
    float(
        y_DEV_FD003_629F.max()
    ),
)

print("\n--- WINDOW FINGERPRINTS ---")

print(
    "FIT windows SHA-256           :",
    FD003_FIT_WINDOW_SHA256_629F,
)

print(
    "FIT targets SHA-256           :",
    FD003_FIT_TARGET_SHA256_629F,
)

print(
    "FIT keys SHA-256              :",
    FD003_FIT_WINDOW_KEYS_SHA256_629F,
)

print(
    "DEV windows SHA-256           :",
    FD003_DEV_WINDOW_SHA256_629F,
)

print(
    "DEV targets SHA-256           :",
    FD003_DEV_TARGET_SHA256_629F,
)

print(
    "DEV keys SHA-256              :",
    FD003_DEV_WINDOW_KEYS_SHA256_629F,
)

print(
    "Window-contract SHA-256       :",
    FD003_WINDOW_CONTRACT_SHA256_629F,
)

print(
    "\nPASS — FD003 FIT/DEV windows are now engine-safe, "
    "30-cycle aligned, fingerprinted, and locked."
)

print(
    "Next: establish the FD003 GRU training architecture and training "
    "protocol before executing model training. TEST features and "
    "official RUL remain unopened."
)

In [ ]:
# STEP 6.29G
# FD003 pre-training historical GRU architecture provenance recovery
# STRICT WHITELIST / NO MODEL EXECUTION / NO TRAINING / NO TEST OR RUL ACCESS

import inspect
import hashlib
import json

import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629f", False)):
    raise RuntimeError(
        "STEP 6.29F must pass before FD003 model-architecture work."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629F", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629F", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

required_629g = [
    "FD002GRURegressor628EB",
    "MODEL_FD002_628EB",
]

missing_629g = [
    name
    for name in required_629g
    if name not in globals()
]

if missing_629g:
    raise RuntimeError(
        "Historical GRU provenance object(s) missing: "
        + ", ".join(missing_629g)
    )

historical_class_629g = FD002GRURegressor628EB
historical_model_629g = MODEL_FD002_628EB

if not isinstance(
    historical_model_629g,
    torch.nn.Module,
):
    raise RuntimeError(
        "MODEL_FD002_628EB is not a torch.nn.Module."
    )

# --------------------------------------------------------------------------------------
# 2) Recover exact historical class signature and source
# --------------------------------------------------------------------------------------

try:
    class_signature_629g = str(
        inspect.signature(
            historical_class_629g
        )
    )
except (TypeError, ValueError):
    class_signature_629g = "<signature unavailable>"

try:
    class_source_629g = inspect.getsource(
        historical_class_629g
    )

    class_source_available_629g = True

    class_source_sha256_629g = hashlib.sha256(
        class_source_629g.encode(
            "utf-8"
        )
    ).hexdigest()

except (OSError, TypeError):

    class_source_629g = "<source unavailable>"
    class_source_available_629g = False
    class_source_sha256_629g = None

# --------------------------------------------------------------------------------------
# 3) Inspect module structure — no forward pass
# --------------------------------------------------------------------------------------

module_rows_629g = []

for module_name_629g, module_629g in (
    historical_model_629g.named_modules()
):

    if module_name_629g == "":
        continue

    row_629g = {
        "module_name":
            module_name_629g,

        "module_class":
            module_629g.__class__.__name__,

        "repr":
            repr(module_629g),
    }

    # Structural attributes only.
    for attr_629g in [
        "input_size",
        "hidden_size",
        "num_layers",
        "dropout",
        "bidirectional",
        "in_features",
        "out_features",
        "p",
    ]:
        if hasattr(
            module_629g,
            attr_629g,
        ):
            value_629g = getattr(
                module_629g,
                attr_629g,
            )

            if isinstance(
                value_629g,
                (
                    str,
                    int,
                    float,
                    bool,
                    type(None),
                ),
            ):
                row_629g[
                    attr_629g
                ] = value_629g

    module_rows_629g.append(
        row_629g
    )


FD003_HISTORICAL_GRU_MODULE_AUDIT_629G = pd.DataFrame(
    module_rows_629g
)

# --------------------------------------------------------------------------------------
# 4) Parameter-shape inventory
# --------------------------------------------------------------------------------------

parameter_rows_629g = []

total_params_629g = 0
trainable_params_629g = 0

for param_name_629g, param_629g in (
    historical_model_629g.named_parameters()
):

    n_param_629g = int(
        param_629g.numel()
    )

    total_params_629g += (
        n_param_629g
    )

    if param_629g.requires_grad:
        trainable_params_629g += (
            n_param_629g
        )

    parameter_rows_629g.append(
        {
            "parameter":
                param_name_629g,

            "shape":
                tuple(
                    param_629g.shape
                ),

            "numel":
                n_param_629g,

            "requires_grad":
                bool(
                    param_629g.requires_grad
                ),

            "dtype":
                str(
                    param_629g.dtype
                ),
        }
    )


FD003_HISTORICAL_GRU_PARAMETER_AUDIT_629G = pd.DataFrame(
    parameter_rows_629g
)

if total_params_629g != 18049:
    raise RuntimeError(
        "Historical model parameter count no longer matches "
        "the transferred 18,049-parameter contract."
    )

# --------------------------------------------------------------------------------------
# 5) Structural fingerprint only
#    Uses module/parameter metadata, NOT weight values.
# --------------------------------------------------------------------------------------

architecture_payload_629g = json.dumps(
    {
        "class_name":
            historical_model_629g.__class__.__name__,

        "class_signature":
            class_signature_629g,

        "modules":
            module_rows_629g,

        "parameters":
            parameter_rows_629g,

        "total_parameters":
            total_params_629g,

        "trainable_parameters":
            trainable_params_629g,
    },
    sort_keys=True,
    default=str,
    separators=(",", ":"),
).encode(
    "utf-8"
)

FD003_HISTORICAL_GRU_ARCHITECTURE_SHA256_629G = (
    hashlib.sha256(
        architecture_payload_629g
    ).hexdigest()
)

FD003_HISTORICAL_GRU_CLASS_SOURCE_SHA256_629G = (
    class_source_sha256_629g
)

FD003_HISTORICAL_GRU_ARCHITECTURE_RECOVERED_629G = True
FD003_MODEL_TRAINING_PERFORMED_629G = False
FD003_TEST_CONTENTS_READ_629G = False
FD003_RUL_CONTENTS_READ_629G = False

all_checks_passed_629g = True

# --------------------------------------------------------------------------------------
# 6) Output
# --------------------------------------------------------------------------------------

print("=" * 126)

print(
    "STEP 6.29G — HISTORICAL GRU ARCHITECTURE "
    "PROVENANCE RECOVERY"
)

print("=" * 126)

print(
    "Model forward pass performed :",
    False,
)

print(
    "Model training performed     :",
    False,
)

print(
    "DEV outcomes inspected       :",
    False,
)

print(
    "TEST contents read           :",
    False,
)

print(
    "Official RUL contents read   :",
    False,
)

print(
    "\n--- HISTORICAL MODEL IDENTITY ---"
)

print(
    "Class name                   :",
    historical_model_629g.__class__.__name__,
)

print(
    "Class signature              :",
    class_signature_629g,
)

print(
    "Parameter count              :",
    total_params_629g,
)

print(
    "Trainable parameter count    :",
    trainable_params_629g,
)

print(
    "Current training mode        :",
    historical_model_629g.training,
)

print(
    "\n--- MODULE STRUCTURE ---"
)

print(
    FD003_HISTORICAL_GRU_MODULE_AUDIT_629G.to_string(
        index=False
    )
)

print(
    "\n--- PARAMETER SHAPES ---"
)

print(
    FD003_HISTORICAL_GRU_PARAMETER_AUDIT_629G.to_string(
        index=False
    )
)

print(
    "\n--- EXACT HISTORICAL CLASS SOURCE ---"
)

print(
    class_source_629g
)

print(
    "\n--- PROVENANCE FINGERPRINTS ---"
)

print(
    "Class source available       :",
    class_source_available_629g,
)

print(
    "Class source SHA-256         :",
    FD003_HISTORICAL_GRU_CLASS_SOURCE_SHA256_629G,
)

print(
    "Architecture SHA-256         :",
    FD003_HISTORICAL_GRU_ARCHITECTURE_SHA256_629G,
)

print(
    "\nPASS — historical GRU architecture provenance recovered "
    "without executing or modifying the model."
)

print(
    "Next: define and freeze the FD003-specific GRU architecture "
    "and training protocol before any training occurs."
)

In [ ]:
# STEP 6.29G-R1
# Historical GRU forward/constructor semantics recovery
# NO MODEL EXECUTION / NO TRAINING / NO DEV SCORING / NO TEST OR RUL ACCESS

import dis
import hashlib
import json
import types

# ---------------------------------------------------------------------
# 1) Preconditions
# ---------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629g", False)):
    raise RuntimeError(
        "STEP 6.29G must pass before forward-semantics recovery."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629G", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629G", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

if "FD002GRURegressor628EB" not in globals():
    raise RuntimeError(
        "Historical FD002GRURegressor628EB class is missing."
    )

historical_class_629gr1 = FD002GRURegressor628EB

# ---------------------------------------------------------------------
# 2) Strictly inspect __init__ and forward only
# ---------------------------------------------------------------------

init_fn_629gr1 = historical_class_629gr1.__init__
forward_fn_629gr1 = historical_class_629gr1.forward

for fn_name_629gr1, fn_629gr1 in [
    ("__init__", init_fn_629gr1),
    ("forward", forward_fn_629gr1),
]:
    if not callable(fn_629gr1):
        raise RuntimeError(
            f"Historical {fn_name_629gr1} is not callable."
        )

    if not hasattr(fn_629gr1, "__code__"):
        raise RuntimeError(
            f"Historical {fn_name_629gr1} has no Python code object."
        )

# ---------------------------------------------------------------------
# 3) Recover code metadata
# ---------------------------------------------------------------------

def _code_metadata_629gr1(fn):
    code = fn.__code__

    return {
        "name": code.co_name,
        "argcount": int(code.co_argcount),
        "posonlyargcount": int(code.co_posonlyargcount),
        "kwonlyargcount": int(code.co_kwonlyargcount),
        "varnames": list(code.co_varnames),
        "names": list(code.co_names),
        "freevars": list(code.co_freevars),
        "cellvars": list(code.co_cellvars),
        "constants_repr": [
            repr(value)
            for value in code.co_consts
        ],
        "bytecode_sha256": hashlib.sha256(
            code.co_code
        ).hexdigest(),
    }


FD003_HISTORICAL_GRU_INIT_CODE_METADATA_629GR1 = (
    _code_metadata_629gr1(
        init_fn_629gr1
    )
)

FD003_HISTORICAL_GRU_FORWARD_CODE_METADATA_629GR1 = (
    _code_metadata_629gr1(
        forward_fn_629gr1
    )
)

# ---------------------------------------------------------------------
# 4) Structured disassembly
# ---------------------------------------------------------------------

def _instruction_rows_629gr1(fn):
    rows = []

    for inst in dis.get_instructions(fn):
        rows.append(
            {
                "offset": int(inst.offset),
                "opname": inst.opname,
                "argrepr": inst.argrepr,
            }
        )

    return rows


FD003_HISTORICAL_GRU_INIT_DISASSEMBLY_629GR1 = (
    _instruction_rows_629gr1(
        init_fn_629gr1
    )
)

FD003_HISTORICAL_GRU_FORWARD_DISASSEMBLY_629GR1 = (
    _instruction_rows_629gr1(
        forward_fn_629gr1
    )
)

# ---------------------------------------------------------------------
# 5) Fingerprint recovered semantics
# ---------------------------------------------------------------------

semantics_payload_629gr1 = json.dumps(
    {
        "class_name":
            historical_class_629gr1.__name__,

        "init_code":
            FD003_HISTORICAL_GRU_INIT_CODE_METADATA_629GR1,

        "forward_code":
            FD003_HISTORICAL_GRU_FORWARD_CODE_METADATA_629GR1,

        "init_disassembly":
            FD003_HISTORICAL_GRU_INIT_DISASSEMBLY_629GR1,

        "forward_disassembly":
            FD003_HISTORICAL_GRU_FORWARD_DISASSEMBLY_629GR1,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_HISTORICAL_GRU_SEMANTICS_SHA256_629GR1 = (
    hashlib.sha256(
        semantics_payload_629gr1
    ).hexdigest()
)

FD003_HISTORICAL_GRU_SEMANTICS_RECOVERED_629GR1 = True
FD003_MODEL_EXECUTION_PERFORMED_629GR1 = False
FD003_MODEL_TRAINING_PERFORMED_629GR1 = False

all_checks_passed_629gr1 = True

# ---------------------------------------------------------------------
# 6) Output
# ---------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29G-R1 — HISTORICAL GRU "
    "FORWARD-SEMANTICS RECOVERY"
)
print("=" * 126)

print("Model forward pass performed :", False)
print("Model training performed     :", False)
print("DEV scoring performed        :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- __init__ CODE METADATA ---")
for key_629gr1, value_629gr1 in (
    FD003_HISTORICAL_GRU_INIT_CODE_METADATA_629GR1.items()
):
    print(
        f"{key_629gr1:20s}:",
        value_629gr1,
    )

print("\n--- __init__ DISASSEMBLY ---")
for row_629gr1 in (
    FD003_HISTORICAL_GRU_INIT_DISASSEMBLY_629GR1
):
    print(
        f"{row_629gr1['offset']:4d}  "
        f"{row_629gr1['opname']:28s} "
        f"{row_629gr1['argrepr']}"
    )

print("\n--- forward CODE METADATA ---")
for key_629gr1, value_629gr1 in (
    FD003_HISTORICAL_GRU_FORWARD_CODE_METADATA_629GR1.items()
):
    print(
        f"{key_629gr1:20s}:",
        value_629gr1,
    )

print("\n--- forward DISASSEMBLY ---")
for row_629gr1 in (
    FD003_HISTORICAL_GRU_FORWARD_DISASSEMBLY_629GR1
):
    print(
        f"{row_629gr1['offset']:4d}  "
        f"{row_629gr1['opname']:28s} "
        f"{row_629gr1['argrepr']}"
    )

print("\n--- SEMANTICS FINGERPRINT ---")
print(
    "Historical GRU semantics SHA-256 :",
    FD003_HISTORICAL_GRU_SEMANTICS_SHA256_629GR1,
)

print(
    "\nPASS — historical GRU constructor/forward semantics "
    "recovered without executing the model."
)

print(
    "Next: define and freeze the FD003-specific 18-input GRU "
    "architecture from this recovered historical implementation; "
    "training will still not occur in that step."
)

In [ ]:
# STEP 6.29H
# FD003-specific GRU architecture definition + structural freeze
# NO MODEL FORWARD PASS / NO TRAINING / NO DEV SCORING / NO TEST OR RUL ACCESS

import hashlib
import json
import dis

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629gr1", False)):
    raise RuntimeError(
        "STEP 6.29G-R1 must pass before defining the FD003 GRU architecture."
    )

required_629h = [
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_WINDOW_CONTRACT_SHA256_629F",
    "FD003_HISTORICAL_GRU_FORWARD_CODE_METADATA_629GR1",
    "FD003_HISTORICAL_GRU_SEMANTICS_SHA256_629GR1",
]

missing_629h = [
    name
    for name in required_629h
    if name not in globals()
]

if missing_629h:
    raise RuntimeError(
        "Missing required FD003 architecture-provenance object(s): "
        + ", ".join(missing_629h)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629G", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629G", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Freeze FD003 architecture hyperparameters
#
# Historical architecture recovered from FD002:
# GRU -> final time-step output -> Linear(64,32) -> ReLU -> Dropout(0.2)
# -> Linear(32,1) -> squeeze(-1)
#
# Dataset-specific change already justified by FD003 preprocessing:
# input_size = 18 instead of historical 17.
# --------------------------------------------------------------------------------------

FD003_GRU_INPUT_SIZE_629H = int(
    len(FD003_NORMALIZED_FEATURE_NAMES_629E)
)

FD003_GRU_HIDDEN_SIZE_629H = 64
FD003_GRU_NUM_LAYERS_629H = 1
FD003_GRU_DENSE_SIZE_629H = 32
FD003_GRU_DROPOUT_629H = 0.2

if FD003_GRU_INPUT_SIZE_629H != 18:
    raise RuntimeError(
        "FD003 frozen preprocessing does not expose exactly 18 model inputs."
    )

# --------------------------------------------------------------------------------------
# 3) Define FD003 model class using recovered historical constructor/forward semantics
# --------------------------------------------------------------------------------------

class FD003GRURegressor629H(nn.Module):
    def __init__(
        self,
        input_size=18,
        hidden_size=64,
        num_layers=1,
        dense_size=32,
        dropout=0.2,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc1 = nn.Linear(
            hidden_size,
            dense_size,
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc2 = nn.Linear(
            dense_size,
            1,
        )

    def forward(self, x):
        gru_output, _ = self.gru(x)
        last_hidden = gru_output[:, -1, :]
        x = self.fc1(last_hidden)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)

# --------------------------------------------------------------------------------------
# 4) Audit-only instantiation with RNG state preserved
#    No persistent model weights are created by this step.
# --------------------------------------------------------------------------------------

with torch.random.fork_rng(
    devices=[],
    enabled=True,
):
    audit_model_629h = FD003GRURegressor629H(
        input_size=FD003_GRU_INPUT_SIZE_629H,
        hidden_size=FD003_GRU_HIDDEN_SIZE_629H,
        num_layers=FD003_GRU_NUM_LAYERS_629H,
        dense_size=FD003_GRU_DENSE_SIZE_629H,
        dropout=FD003_GRU_DROPOUT_629H,
    )

    parameter_rows_629h = []

    total_params_629h = 0
    trainable_params_629h = 0

    for param_name_629h, param_629h in audit_model_629h.named_parameters():
        n_param_629h = int(
            param_629h.numel()
        )

        total_params_629h += n_param_629h

        if param_629h.requires_grad:
            trainable_params_629h += n_param_629h

        parameter_rows_629h.append(
            {
                "parameter": param_name_629h,
                "shape": tuple(param_629h.shape),
                "numel": n_param_629h,
                "requires_grad": bool(param_629h.requires_grad),
                "dtype": str(param_629h.dtype),
            }
        )

    module_rows_629h = []

    for module_name_629h, module_629h in audit_model_629h.named_modules():
        if module_name_629h == "":
            continue

        row_629h = {
            "module_name": module_name_629h,
            "module_class": module_629h.__class__.__name__,
            "repr": repr(module_629h),
        }

        for attr_629h in [
            "input_size",
            "hidden_size",
            "num_layers",
            "dropout",
            "bidirectional",
            "in_features",
            "out_features",
            "p",
        ]:
            if hasattr(module_629h, attr_629h):
                value_629h = getattr(
                    module_629h,
                    attr_629h,
                )

                if isinstance(
                    value_629h,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                ):
                    row_629h[attr_629h] = value_629h

        module_rows_629h.append(
            row_629h
        )

del audit_model_629h

FD003_GRU_PARAMETER_AUDIT_629H = pd.DataFrame(
    parameter_rows_629h
)

FD003_GRU_MODULE_AUDIT_629H = pd.DataFrame(
    module_rows_629h
)

# --------------------------------------------------------------------------------------
# 5) Exact structural checks
# --------------------------------------------------------------------------------------

expected_parameter_shapes_629h = {
    "gru.weight_ih_l0": (192, 18),
    "gru.weight_hh_l0": (192, 64),
    "gru.bias_ih_l0": (192,),
    "gru.bias_hh_l0": (192,),
    "fc1.weight": (32, 64),
    "fc1.bias": (32,),
    "fc2.weight": (1, 32),
    "fc2.bias": (1,),
}

actual_parameter_shapes_629h = {
    row["parameter"]: tuple(row["shape"])
    for row in parameter_rows_629h
}

if actual_parameter_shapes_629h != expected_parameter_shapes_629h:
    raise RuntimeError(
        "FD003 GRU parameter-shape contract does not match "
        "the recovered architecture."
    )

FD003_GRU_EXPECTED_PARAMETER_COUNT_629H = 18241

if total_params_629h != FD003_GRU_EXPECTED_PARAMETER_COUNT_629H:
    raise RuntimeError(
        "FD003 GRU parameter count mismatch: "
        f"expected {FD003_GRU_EXPECTED_PARAMETER_COUNT_629H}, "
        f"found {total_params_629h}."
    )

if trainable_params_629h != FD003_GRU_EXPECTED_PARAMETER_COUNT_629H:
    raise RuntimeError(
        "FD003 GRU trainable parameter count mismatch."
    )

# --------------------------------------------------------------------------------------
# 6) Confirm the new forward method has the recovered semantic operations
# --------------------------------------------------------------------------------------

forward_meta_629h = {
    "varnames": list(
        FD003GRURegressor629H.forward.__code__.co_varnames
    ),
    "names": list(
        FD003GRURegressor629H.forward.__code__.co_names
    ),
    "constants_repr": [
        repr(value_629h)
        for value_629h
        in FD003GRURegressor629H.forward.__code__.co_consts
    ],
}

required_forward_names_629h = [
    "gru",
    "fc1",
    "relu",
    "dropout",
    "fc2",
    "squeeze",
]

forward_names_exact_629h = bool(
    forward_meta_629h["names"]
    ==
    required_forward_names_629h
)

if not forward_names_exact_629h:
    raise RuntimeError(
        "FD003 forward operation sequence differs from the "
        "recovered historical semantics."
    )

forward_instruction_names_629h = [
    inst.opname
    for inst in dis.get_instructions(
        FD003GRURegressor629H.forward
    )
]

if "BINARY_SUBSCR" not in forward_instruction_names_629h:
    raise RuntimeError(
        "FD003 forward semantics do not contain final-time-step indexing."
    )

# --------------------------------------------------------------------------------------
# 7) Freeze architecture contract
# --------------------------------------------------------------------------------------

FD003_GRU_ARCHITECTURE_CONTRACT_629H = {
    "step": "6.29H",
    "dataset": "FD003",

    "historical_family":
        "FD002GRURegressor628EB",

    "historical_semantics_sha256":
        FD003_HISTORICAL_GRU_SEMANTICS_SHA256_629GR1,

    "dataset_specific_change":
        "input_size changed from historical 17 to FD003-locked 18 only",

    "input_size":
        FD003_GRU_INPUT_SIZE_629H,

    "hidden_size":
        FD003_GRU_HIDDEN_SIZE_629H,

    "num_layers":
        FD003_GRU_NUM_LAYERS_629H,

    "batch_first":
        True,

    "dense_size":
        FD003_GRU_DENSE_SIZE_629H,

    "activation":
        "ReLU",

    "dropout":
        FD003_GRU_DROPOUT_629H,

    "output_size":
        1,

    "forward_semantics":
        "GRU full sequence -> final time-step output -> "
        "fc1 -> ReLU -> dropout -> fc2 -> squeeze(-1)",

    "parameter_count":
        total_params_629h,

    "trainable_parameter_count":
        trainable_params_629h,

    "parameter_shapes":
        {
            key: list(value)
            for key, value in expected_parameter_shapes_629h.items()
        },

    "window_contract_sha256":
        FD003_WINDOW_CONTRACT_SHA256_629F,

    "model_forward_executed":
        False,

    "model_training_performed":
        False,

    "dev_scoring_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

architecture_payload_629h = json.dumps(
    FD003_GRU_ARCHITECTURE_CONTRACT_629H,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H = (
    hashlib.sha256(
        architecture_payload_629h
    ).hexdigest()
)

FD003_GRU_ARCHITECTURE_LOCKED_629H = True
FD003_MODEL_INSTANCE_CREATED_629H = False
FD003_MODEL_TRAINING_PERFORMED_629H = False
FD003_TEST_CONTENTS_READ_629H = False
FD003_RUL_CONTENTS_READ_629H = False

all_checks_passed_629h = True

# --------------------------------------------------------------------------------------
# 8) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29H — FD003-SPECIFIC GRU ARCHITECTURE "
    "DEFINITION + STRUCTURAL FREEZE"
)
print("=" * 126)

print("Model forward pass performed :", False)
print("Model training performed     :", False)
print("Persistent model instantiated:", False)
print("DEV scoring performed        :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- LOCKED FD003 ARCHITECTURE ---")
print(
    "Input size                   :",
    FD003_GRU_INPUT_SIZE_629H,
)
print(
    "Hidden size                  :",
    FD003_GRU_HIDDEN_SIZE_629H,
)
print(
    "GRU layers                   :",
    FD003_GRU_NUM_LAYERS_629H,
)
print(
    "Dense size                   :",
    FD003_GRU_DENSE_SIZE_629H,
)
print(
    "Dropout                      :",
    FD003_GRU_DROPOUT_629H,
)
print(
    "Output size                  :",
    1,
)
print(
    "Parameter count              :",
    total_params_629h,
)
print(
    "Trainable parameter count    :",
    trainable_params_629h,
)
print(
    "Historical parameter count   :",
    18049,
)
print(
    "Expected +192 input params   :",
    total_params_629h - 18049,
)

print("\n--- MODULE STRUCTURE ---")
print(
    FD003_GRU_MODULE_AUDIT_629H.to_string(
        index=False
    )
)

print("\n--- PARAMETER SHAPES ---")
print(
    FD003_GRU_PARAMETER_AUDIT_629H.to_string(
        index=False
    )
)

print("\n--- FORWARD SEMANTICS CHECK ---")
print(
    "Forward names exact          :",
    forward_names_exact_629h,
)
print(
    "Forward names                :",
    forward_meta_629h["names"],
)
print(
    "Final-time-step indexing     :",
    True,
)

print("\n--- ARCHITECTURE PROVENANCE ---")
print(
    "Historical semantics SHA-256 :",
    FD003_HISTORICAL_GRU_SEMANTICS_SHA256_629GR1,
)
print(
    "Architecture-contract SHA-256:",
    FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H,
)

print(
    "\nPASS — FD003 GRU architecture is now structurally frozen. "
    "The only dataset-specific architecture change is input_size 17 -> 18."
)

print(
    "Next: recover/define and freeze the FD003 training protocol "
    "(optimizer, learning rate, batching, epoch/early-stopping rules, "
    "training seed) before creating the persistent model or training it."
)

In [ ]:
# STEP 6.29I
# FD003 prospective training-protocol freeze
# NO MODEL INSTANTIATION / NO TRAINING / NO DEV SCORING / NO TEST OR RUL ACCESS

import hashlib
import json

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629h", False)):
    raise RuntimeError(
        "STEP 6.29H must pass before freezing the FD003 training protocol."
    )

required_629i = [
    "FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H",
    "FD003_WINDOW_CONTRACT_SHA256_629F",
    "FD003_SPLIT_CONTRACT_SHA256_629C",
]

missing_629i = [
    name
    for name in required_629i
    if name not in globals()
]

if missing_629i:
    raise RuntimeError(
        "Missing required FD003 pre-training contract object(s): "
        + ", ".join(missing_629i)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629H", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629H", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

if bool(globals().get("FD003_MODEL_TRAINING_PERFORMED_629H", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates model training already occurred."
    )

# --------------------------------------------------------------------------------------
# 2) Provenance statement
# --------------------------------------------------------------------------------------

FD003_TRAINING_PROTOCOL_PROVENANCE_629I = (
    "prospectively fixed for FD003 before any FD003 model training or DEV model score; "
    "historical FD002 optimizer/LR/epoch/early-stopping values were not provenance-"
    "recovered strongly enough to claim exact reuse"
)

# --------------------------------------------------------------------------------------
# 3) Freeze training protocol
# --------------------------------------------------------------------------------------

FD003_TRAINING_SEED_629I = 42

FD003_TRAIN_BATCH_SIZE_629I = 256
FD003_EVAL_BATCH_SIZE_629I = 512

FD003_OPTIMIZER_629I = "Adam"
FD003_LEARNING_RATE_629I = 1e-3
FD003_ADAM_BETAS_629I = (0.9, 0.999)
FD003_ADAM_EPS_629I = 1e-8
FD003_WEIGHT_DECAY_629I = 0.0

FD003_TRAIN_LOSS_629I = "MSELoss"

FD003_MAX_EPOCHS_629I = 100
FD003_EARLY_STOPPING_MONITOR_629I = "DEV_MAE"
FD003_EARLY_STOPPING_MODE_629I = "min"
FD003_EARLY_STOPPING_PATIENCE_629I = 10
FD003_EARLY_STOPPING_MIN_DELTA_629I = 0.0
FD003_RESTORE_BEST_WEIGHTS_629I = True
FD003_TIE_BREAK_RULE_629I = "earliest epoch"

FD003_FIT_SHUFFLE_629I = True
FD003_SINGLE_TRAINING_RUN_629I = True
FD003_HYPERPARAMETER_SEARCH_PERMITTED_629I = False
FD003_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_629I = False

FD003_LR_SCHEDULER_629I = None
FD003_GRADIENT_CLIPPING_629I = None

# --------------------------------------------------------------------------------------
# 4) Freeze deterministic execution conventions
# --------------------------------------------------------------------------------------

FD003_RNG_PROTOCOL_629I = {
    "python_random_seed": FD003_TRAINING_SEED_629I,
    "numpy_seed": FD003_TRAINING_SEED_629I,
    "torch_cpu_seed": FD003_TRAINING_SEED_629I,
    "torch_cuda_seed_all_if_available": FD003_TRAINING_SEED_629I,
    "fit_dataloader_generator_seed": FD003_TRAINING_SEED_629I,
}

# --------------------------------------------------------------------------------------
# 5) Training protocol contract
# --------------------------------------------------------------------------------------

FD003_TRAINING_PROTOCOL_CONTRACT_629I = {
    "step": "6.29I",
    "dataset": "FD003",

    "provenance":
        FD003_TRAINING_PROTOCOL_PROVENANCE_629I,

    "architecture_contract_sha256":
        FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H,

    "window_contract_sha256":
        FD003_WINDOW_CONTRACT_SHA256_629F,

    "split_contract_sha256":
        FD003_SPLIT_CONTRACT_SHA256_629C,

    "training_seed":
        FD003_TRAINING_SEED_629I,

    "train_batch_size":
        FD003_TRAIN_BATCH_SIZE_629I,

    "eval_batch_size":
        FD003_EVAL_BATCH_SIZE_629I,

    "optimizer":
        FD003_OPTIMIZER_629I,

    "learning_rate":
        FD003_LEARNING_RATE_629I,

    "adam_betas":
        list(FD003_ADAM_BETAS_629I),

    "adam_eps":
        FD003_ADAM_EPS_629I,

    "weight_decay":
        FD003_WEIGHT_DECAY_629I,

    "training_loss":
        FD003_TRAIN_LOSS_629I,

    "max_epochs":
        FD003_MAX_EPOCHS_629I,

    "early_stopping_monitor":
        FD003_EARLY_STOPPING_MONITOR_629I,

    "early_stopping_mode":
        FD003_EARLY_STOPPING_MODE_629I,

    "early_stopping_patience":
        FD003_EARLY_STOPPING_PATIENCE_629I,

    "early_stopping_min_delta":
        FD003_EARLY_STOPPING_MIN_DELTA_629I,

    "restore_best_weights":
        FD003_RESTORE_BEST_WEIGHTS_629I,

    "tie_break_rule":
        FD003_TIE_BREAK_RULE_629I,

    "fit_shuffle":
        FD003_FIT_SHUFFLE_629I,

    "single_training_run":
        FD003_SINGLE_TRAINING_RUN_629I,

    "hyperparameter_search_permitted":
        FD003_HYPERPARAMETER_SEARCH_PERMITTED_629I,

    "dev_based_retuning_after_run_permitted":
        FD003_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_629I,

    "lr_scheduler":
        FD003_LR_SCHEDULER_629I,

    "gradient_clipping":
        FD003_GRADIENT_CLIPPING_629I,

    "rng_protocol":
        dict(FD003_RNG_PROTOCOL_629I),

    "persistent_model_created":
        False,

    "model_training_performed":
        False,

    "dev_model_score_inspected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629i = json.dumps(
    FD003_TRAINING_PROTOCOL_CONTRACT_629I,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I = hashlib.sha256(
    contract_payload_629i
).hexdigest()

FD003_TRAINING_PROTOCOL_LOCKED_629I = True
FD003_MODEL_INSTANCE_CREATED_629I = False
FD003_MODEL_TRAINING_PERFORMED_629I = False
FD003_DEV_MODEL_SCORE_INSPECTED_629I = False
FD003_TEST_CONTENTS_READ_629I = False
FD003_RUL_CONTENTS_READ_629I = False

all_checks_passed_629i = True

# --------------------------------------------------------------------------------------
# 6) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29I — FD003 PROSPECTIVE TRAINING-PROTOCOL FREEZE"
)
print("=" * 126)

print("Persistent model created     :", False)
print("Model training performed     :", False)
print("DEV model score inspected    :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- PROVENANCE ---")
print(
    FD003_TRAINING_PROTOCOL_PROVENANCE_629I
)

print("\n--- LOCKED TRAINING PROTOCOL ---")
print("Training seed                :", FD003_TRAINING_SEED_629I)
print("Training loss                :", FD003_TRAIN_LOSS_629I)
print("Optimizer                    :", FD003_OPTIMIZER_629I)
print("Learning rate                :", FD003_LEARNING_RATE_629I)
print("Adam betas                   :", FD003_ADAM_BETAS_629I)
print("Adam eps                     :", FD003_ADAM_EPS_629I)
print("Weight decay                 :", FD003_WEIGHT_DECAY_629I)
print("FIT batch size               :", FD003_TRAIN_BATCH_SIZE_629I)
print("Evaluation batch size        :", FD003_EVAL_BATCH_SIZE_629I)
print("FIT shuffle                  :", FD003_FIT_SHUFFLE_629I)
print("Maximum epochs               :", FD003_MAX_EPOCHS_629I)
print("Early-stop monitor           :", FD003_EARLY_STOPPING_MONITOR_629I)
print("Early-stop mode              :", FD003_EARLY_STOPPING_MODE_629I)
print("Early-stop patience          :", FD003_EARLY_STOPPING_PATIENCE_629I)
print("Early-stop minimum delta     :", FD003_EARLY_STOPPING_MIN_DELTA_629I)
print("Restore best weights         :", FD003_RESTORE_BEST_WEIGHTS_629I)
print("Tie-break rule               :", FD003_TIE_BREAK_RULE_629I)
print("LR scheduler                 :", FD003_LR_SCHEDULER_629I)
print("Gradient clipping            :", FD003_GRADIENT_CLIPPING_629I)

print("\n--- ANTI-RETUNING RULES ---")
print("Single prospective run       :", FD003_SINGLE_TRAINING_RUN_629I)
print(
    "Hyperparameter search allowed:",
    FD003_HYPERPARAMETER_SEARCH_PERMITTED_629I,
)
print(
    "Retune after DEV run allowed :",
    FD003_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_629I,
)

print(
    "\nTraining-protocol SHA-256    :",
    FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I,
)

print(
    "\nPASS — FD003 training protocol is frozen prospectively before "
    "persistent model creation, model training, or DEV model scoring."
)

print(
    "Next: create the deterministic FIT/DEV DataLoaders and persistent "
    "FD003 model/optimizer under this locked protocol, but do not train "
    "until their initialization fingerprints are recorded."
)

In [ ]:
# STEP 6.29J
# FD003 deterministic DataLoader + persistent model/optimizer initialization lock
# NO TRAINING / NO FORWARD PASS / NO DEV SCORING / NO TEST OR RUL ACCESS

import hashlib
import json
import random

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629i", False)):
    raise RuntimeError(
        "STEP 6.29I must pass before FD003 persistent model/optimizer initialization."
    )

required_629j = [
    "FD003GRURegressor629H",
    "X_FIT_FD003_629F",
    "y_FIT_FD003_629F",
    "X_DEV_FD003_629F",
    "y_DEV_FD003_629F",
    "FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H",
    "FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I",
    "FD003_TRAINING_SEED_629I",
    "FD003_TRAIN_BATCH_SIZE_629I",
    "FD003_EVAL_BATCH_SIZE_629I",
    "FD003_LEARNING_RATE_629I",
    "FD003_ADAM_BETAS_629I",
    "FD003_ADAM_EPS_629I",
    "FD003_WEIGHT_DECAY_629I",
]

missing_629j = [
    name
    for name in required_629j
    if name not in globals()
]

if missing_629j:
    raise RuntimeError(
        "Missing required FD003 initialization object(s): "
        + ", ".join(missing_629j)
    )

if bool(globals().get("FD003_MODEL_INSTANCE_CREATED_629J", False)):
    raise RuntimeError(
        "STEP 6.29J has already created the persistent FD003 model in this session."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629I", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629I", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

if bool(globals().get("FD003_MODEL_TRAINING_PERFORMED_629I", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates model training already occurred."
    )

# --------------------------------------------------------------------------------------
# 2) Reproducible RNG initialization
# --------------------------------------------------------------------------------------

random.seed(
    FD003_TRAINING_SEED_629I
)

np.random.seed(
    FD003_TRAINING_SEED_629I
)

torch.manual_seed(
    FD003_TRAINING_SEED_629I
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        FD003_TRAINING_SEED_629I
    )

FD003_DEVICE_629J = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FD003_FIT_LOADER_GENERATOR_629J = torch.Generator()
FD003_FIT_LOADER_GENERATOR_629J.manual_seed(
    FD003_TRAINING_SEED_629I
)

# --------------------------------------------------------------------------------------
# 3) TensorDataset objects from already-frozen window tensors
# --------------------------------------------------------------------------------------

fit_X_tensor_629j = torch.from_numpy(
    np.asarray(
        X_FIT_FD003_629F,
        dtype=np.float32,
    )
)

fit_y_tensor_629j = torch.from_numpy(
    np.asarray(
        y_FIT_FD003_629F,
        dtype=np.float32,
    )
)

dev_X_tensor_629j = torch.from_numpy(
    np.asarray(
        X_DEV_FD003_629F,
        dtype=np.float32,
    )
)

dev_y_tensor_629j = torch.from_numpy(
    np.asarray(
        y_DEV_FD003_629F,
        dtype=np.float32,
    )
)

if tuple(fit_X_tensor_629j.shape) != (17114, 30, 18):
    raise RuntimeError(
        "FD003 FIT tensor shape changed before DataLoader creation."
    )

if tuple(dev_X_tensor_629j.shape) != (4706, 30, 18):
    raise RuntimeError(
        "FD003 DEV tensor shape changed before DataLoader creation."
    )

FD003_FIT_DATASET_629J = TensorDataset(
    fit_X_tensor_629j,
    fit_y_tensor_629j,
)

FD003_DEV_DATASET_629J = TensorDataset(
    dev_X_tensor_629j,
    dev_y_tensor_629j,
)

FD003_FIT_LOADER_629J = DataLoader(
    FD003_FIT_DATASET_629J,
    batch_size=FD003_TRAIN_BATCH_SIZE_629I,
    shuffle=True,
    generator=FD003_FIT_LOADER_GENERATOR_629J,
    num_workers=0,
    drop_last=False,
)

FD003_DEV_LOADER_629J = DataLoader(
    FD003_DEV_DATASET_629J,
    batch_size=FD003_EVAL_BATCH_SIZE_629I,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

# --------------------------------------------------------------------------------------
# 4) Persistent model, loss, optimizer
#    Creation only — no forward pass and no optimizer step.
# --------------------------------------------------------------------------------------

MODEL_FD003_629J = FD003GRURegressor629H(
    input_size=18,
    hidden_size=64,
    num_layers=1,
    dense_size=32,
    dropout=0.2,
)

MODEL_FD003_629J.to(
    FD003_DEVICE_629J
)

MODEL_FD003_629J.train()

FD003_LOSS_FN_629J = torch.nn.MSELoss()

OPTIMIZER_FD003_629J = torch.optim.Adam(
    MODEL_FD003_629J.parameters(),
    lr=FD003_LEARNING_RATE_629I,
    betas=FD003_ADAM_BETAS_629I,
    eps=FD003_ADAM_EPS_629I,
    weight_decay=FD003_WEIGHT_DECAY_629I,
)

# --------------------------------------------------------------------------------------
# 5) Stable initial model-state fingerprint
# --------------------------------------------------------------------------------------

def _state_dict_sha256_629j(model):
    hasher = hashlib.sha256()

    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes = key.encode("utf-8")
        dtype_bytes = str(tensor.dtype).encode("utf-8")
        shape_bytes = json.dumps(
            list(tensor.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(
            len(key_bytes).to_bytes(8, "little")
        )
        hasher.update(key_bytes)

        hasher.update(
            len(dtype_bytes).to_bytes(8, "little")
        )
        hasher.update(dtype_bytes)

        hasher.update(
            len(shape_bytes).to_bytes(8, "little")
        )
        hasher.update(shape_bytes)

        hasher.update(
            tensor.numpy().tobytes(order="C")
        )

    return hasher.hexdigest()


FD003_INITIAL_MODEL_STATE_SHA256_629J = (
    _state_dict_sha256_629j(
        MODEL_FD003_629J
    )
)

# --------------------------------------------------------------------------------------
# 6) Optimizer initialization fingerprint
# --------------------------------------------------------------------------------------

optimizer_state_629j = OPTIMIZER_FD003_629J.state_dict()

if len(optimizer_state_629j["state"]) != 0:
    raise RuntimeError(
        "Adam optimizer unexpectedly has non-empty state before training."
    )

optimizer_config_629j = {
    "optimizer_class":
        OPTIMIZER_FD003_629J.__class__.__name__,
    "lr":
        float(FD003_LEARNING_RATE_629I),
    "betas":
        [
            float(FD003_ADAM_BETAS_629I[0]),
            float(FD003_ADAM_BETAS_629I[1]),
        ],
    "eps":
        float(FD003_ADAM_EPS_629I),
    "weight_decay":
        float(FD003_WEIGHT_DECAY_629I),
    "parameter_group_count":
        int(len(optimizer_state_629j["param_groups"])),
    "optimizer_state_entry_count":
        int(len(optimizer_state_629j["state"])),
}

optimizer_payload_629j = json.dumps(
    optimizer_config_629j,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_INITIAL_OPTIMIZER_CONFIG_SHA256_629J = hashlib.sha256(
    optimizer_payload_629j
).hexdigest()

# --------------------------------------------------------------------------------------
# 7) Initialization contract
# --------------------------------------------------------------------------------------

FD003_INITIALIZATION_CONTRACT_629J = {
    "step": "6.29J",
    "dataset": "FD003",
    "architecture_contract_sha256":
        FD003_GRU_ARCHITECTURE_CONTRACT_SHA256_629H,
    "training_protocol_contract_sha256":
        FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I,
    "device":
        str(FD003_DEVICE_629J),
    "torch_version":
        str(torch.__version__),
    "training_seed":
        int(FD003_TRAINING_SEED_629I),
    "fit_dataset_size":
        int(len(FD003_FIT_DATASET_629J)),
    "dev_dataset_size":
        int(len(FD003_DEV_DATASET_629J)),
    "fit_batch_size":
        int(FD003_TRAIN_BATCH_SIZE_629I),
    "dev_batch_size":
        int(FD003_EVAL_BATCH_SIZE_629I),
    "fit_batch_count":
        int(len(FD003_FIT_LOADER_629J)),
    "dev_batch_count":
        int(len(FD003_DEV_LOADER_629J)),
    "fit_shuffle":
        True,
    "fit_loader_generator_seed":
        int(FD003_TRAINING_SEED_629I),
    "num_workers":
        0,
    "drop_last":
        False,
    "loss_function":
        FD003_LOSS_FN_629J.__class__.__name__,
    "optimizer":
        optimizer_config_629j,
    "initial_model_state_sha256":
        FD003_INITIAL_MODEL_STATE_SHA256_629J,
    "initial_optimizer_config_sha256":
        FD003_INITIAL_OPTIMIZER_CONFIG_SHA256_629J,
    "model_forward_executed":
        False,
    "optimizer_step_executed":
        False,
    "model_training_performed":
        False,
    "dev_scoring_performed":
        False,
    "test_contents_read":
        False,
    "official_rul_contents_read":
        False,
}

initialization_payload_629j = json.dumps(
    FD003_INITIALIZATION_CONTRACT_629J,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_INITIALIZATION_CONTRACT_SHA256_629J = hashlib.sha256(
    initialization_payload_629j
).hexdigest()

FD003_MODEL_INSTANCE_CREATED_629J = True
FD003_OPTIMIZER_CREATED_629J = True
FD003_DATALOADERS_CREATED_629J = True

FD003_MODEL_FORWARD_EXECUTED_629J = False
FD003_OPTIMIZER_STEP_EXECUTED_629J = False
FD003_MODEL_TRAINING_PERFORMED_629J = False
FD003_DEV_SCORING_PERFORMED_629J = False
FD003_TEST_CONTENTS_READ_629J = False
FD003_RUL_CONTENTS_READ_629J = False

all_checks_passed_629j = True

# --------------------------------------------------------------------------------------
# 8) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29J — FD003 DATALOADER + PERSISTENT MODEL/OPTIMIZER "
    "INITIALIZATION LOCK"
)
print("=" * 126)

print("Model forward pass performed :", False)
print("Optimizer step performed     :", False)
print("Model training performed     :", False)
print("DEV scoring performed        :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- EXECUTION ENVIRONMENT ---")
print("Device                       :", FD003_DEVICE_629J)
print("Torch version                :", torch.__version__)
print("Training seed                :", FD003_TRAINING_SEED_629I)

print("\n--- DATALOADERS ---")
print("FIT dataset size             :", len(FD003_FIT_DATASET_629J))
print("DEV dataset size             :", len(FD003_DEV_DATASET_629J))
print("FIT batch size               :", FD003_TRAIN_BATCH_SIZE_629I)
print("DEV batch size               :", FD003_EVAL_BATCH_SIZE_629I)
print("FIT batch count              :", len(FD003_FIT_LOADER_629J))
print("DEV batch count              :", len(FD003_DEV_LOADER_629J))
print("FIT shuffle                  :", True)
print("DataLoader workers           :", 0)

print("\n--- PERSISTENT MODEL / OPTIMIZER ---")
print(
    "Model class                  :",
    MODEL_FD003_629J.__class__.__name__,
)
print(
    "Model parameter count        :",
    sum(
        p.numel()
        for p in MODEL_FD003_629J.parameters()
    ),
)
print(
    "Model training mode          :",
    MODEL_FD003_629J.training,
)
print(
    "Loss function                :",
    FD003_LOSS_FN_629J.__class__.__name__,
)
print(
    "Optimizer                    :",
    OPTIMIZER_FD003_629J.__class__.__name__,
)
print(
    "Optimizer state entries      :",
    len(
        OPTIMIZER_FD003_629J.state_dict()["state"]
    ),
)

print("\n--- INITIALIZATION FINGERPRINTS ---")
print(
    "Initial model-state SHA-256  :",
    FD003_INITIAL_MODEL_STATE_SHA256_629J,
)
print(
    "Optimizer-config SHA-256     :",
    FD003_INITIAL_OPTIMIZER_CONFIG_SHA256_629J,
)
print(
    "Initialization-contract SHA  :",
    FD003_INITIALIZATION_CONTRACT_SHA256_629J,
)

print(
    "\nPASS — deterministic FD003 DataLoaders, persistent model, loss, "
    "and optimizer are initialized and fingerprinted before any training."
)

print(
    "Next: execute the single locked FD003 training run with DEV_MAE "
    "early stopping, restore the best epoch, and fingerprint the trained "
    "model. TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29J-R1
# FD003 writable-backing DataLoader safety correction
# NO MODEL REINITIALIZATION / NO TRAINING / NO FORWARD PASS / NO DEV SCORING / NO TEST OR RUL ACCESS

import hashlib
import json
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629j", False)):
    raise RuntimeError(
        "STEP 6.29J must pass before the writable-backing DataLoader correction."
    )

required_629jr1 = [
    "X_FIT_FD003_629F",
    "y_FIT_FD003_629F",
    "X_DEV_FD003_629F",
    "y_DEV_FD003_629F",
    "MODEL_FD003_629J",
    "OPTIMIZER_FD003_629J",
    "FD003_INITIAL_MODEL_STATE_SHA256_629J",
    "FD003_INITIAL_OPTIMIZER_CONFIG_SHA256_629J",
    "FD003_INITIALIZATION_CONTRACT_SHA256_629J",
    "FD003_TRAINING_SEED_629I",
    "FD003_TRAIN_BATCH_SIZE_629I",
    "FD003_EVAL_BATCH_SIZE_629I",
]

missing_629jr1 = [
    name
    for name in required_629jr1
    if name not in globals()
]

if missing_629jr1:
    raise RuntimeError(
        "Missing required STEP 6.29J object(s): "
        + ", ".join(missing_629jr1)
    )

if bool(globals().get("FD003_MODEL_TRAINING_PERFORMED_629J", True)):
    raise RuntimeError(
        "Training has already occurred; do not rewrite the input pipeline now."
    )

if bool(globals().get("FD003_MODEL_FORWARD_EXECUTED_629J", True)):
    raise RuntimeError(
        "A model forward pass has already occurred; stop before correction."
    )

if bool(globals().get("FD003_OPTIMIZER_STEP_EXECUTED_629J", True)):
    raise RuntimeError(
        "An optimizer step has already occurred; stop before correction."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629J", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629J", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Raw-byte helper
# --------------------------------------------------------------------------------------

def _raw_sha256_629jr1(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during writable-copy verification."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()

# --------------------------------------------------------------------------------------
# 3) Create explicit writable C-contiguous copies
# --------------------------------------------------------------------------------------

FIT_X_WRITABLE_FD003_629JR1 = np.array(
    X_FIT_FD003_629F,
    dtype=np.float32,
    order="C",
    copy=True,
)

FIT_Y_WRITABLE_FD003_629JR1 = np.array(
    y_FIT_FD003_629F,
    dtype=np.float32,
    order="C",
    copy=True,
)

DEV_X_WRITABLE_FD003_629JR1 = np.array(
    X_DEV_FD003_629F,
    dtype=np.float32,
    order="C",
    copy=True,
)

DEV_Y_WRITABLE_FD003_629JR1 = np.array(
    y_DEV_FD003_629F,
    dtype=np.float32,
    order="C",
    copy=True,
)

for name_629jr1, arr_629jr1 in [
    ("FIT_X", FIT_X_WRITABLE_FD003_629JR1),
    ("FIT_Y", FIT_Y_WRITABLE_FD003_629JR1),
    ("DEV_X", DEV_X_WRITABLE_FD003_629JR1),
    ("DEV_Y", DEV_Y_WRITABLE_FD003_629JR1),
]:
    if not arr_629jr1.flags.writeable:
        raise RuntimeError(
            f"{name_629jr1} writable copy is unexpectedly read-only."
        )

    if not arr_629jr1.flags.c_contiguous:
        raise RuntimeError(
            f"{name_629jr1} writable copy is unexpectedly non-contiguous."
        )

# Exact-content verification against the frozen window/target arrays.
content_identity_629jr1 = {
    "fit_x": bool(
        _raw_sha256_629jr1(FIT_X_WRITABLE_FD003_629JR1)
        ==
        _raw_sha256_629jr1(X_FIT_FD003_629F)
    ),

    "fit_y": bool(
        _raw_sha256_629jr1(FIT_Y_WRITABLE_FD003_629JR1)
        ==
        _raw_sha256_629jr1(y_FIT_FD003_629F)
    ),

    "dev_x": bool(
        _raw_sha256_629jr1(DEV_X_WRITABLE_FD003_629JR1)
        ==
        _raw_sha256_629jr1(X_DEV_FD003_629F)
    ),

    "dev_y": bool(
        _raw_sha256_629jr1(DEV_Y_WRITABLE_FD003_629JR1)
        ==
        _raw_sha256_629jr1(y_DEV_FD003_629F)
    ),
}

if not all(content_identity_629jr1.values()):
    raise RuntimeError(
        "Writable-copy content identity check failed."
    )

# --------------------------------------------------------------------------------------
# 4) Build safe TensorDatasets / DataLoaders
# --------------------------------------------------------------------------------------

FIT_X_TENSOR_FD003_629JR1 = torch.from_numpy(
    FIT_X_WRITABLE_FD003_629JR1
)

FIT_Y_TENSOR_FD003_629JR1 = torch.from_numpy(
    FIT_Y_WRITABLE_FD003_629JR1
)

DEV_X_TENSOR_FD003_629JR1 = torch.from_numpy(
    DEV_X_WRITABLE_FD003_629JR1
)

DEV_Y_TENSOR_FD003_629JR1 = torch.from_numpy(
    DEV_Y_WRITABLE_FD003_629JR1
)

FD003_FIT_DATASET_629JR1 = TensorDataset(
    FIT_X_TENSOR_FD003_629JR1,
    FIT_Y_TENSOR_FD003_629JR1,
)

FD003_DEV_DATASET_629JR1 = TensorDataset(
    DEV_X_TENSOR_FD003_629JR1,
    DEV_Y_TENSOR_FD003_629JR1,
)

FD003_FIT_LOADER_GENERATOR_629JR1 = torch.Generator()

FD003_FIT_LOADER_GENERATOR_629JR1.manual_seed(
    FD003_TRAINING_SEED_629I
)

FD003_FIT_LOADER_629JR1 = DataLoader(
    FD003_FIT_DATASET_629JR1,
    batch_size=FD003_TRAIN_BATCH_SIZE_629I,
    shuffle=True,
    generator=FD003_FIT_LOADER_GENERATOR_629JR1,
    num_workers=0,
    drop_last=False,
)

FD003_DEV_LOADER_629JR1 = DataLoader(
    FD003_DEV_DATASET_629JR1,
    batch_size=FD003_EVAL_BATCH_SIZE_629I,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

# --------------------------------------------------------------------------------------
# 5) Verify model / optimizer initialization remained untouched
# --------------------------------------------------------------------------------------

def _state_dict_sha256_629jr1(model):
    hasher = hashlib.sha256()

    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes = key.encode("utf-8")

        dtype_bytes = str(
            tensor.dtype
        ).encode("utf-8")

        shape_bytes = json.dumps(
            list(tensor.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(
            len(key_bytes).to_bytes(8, "little")
        )

        hasher.update(key_bytes)

        hasher.update(
            len(dtype_bytes).to_bytes(8, "little")
        )

        hasher.update(dtype_bytes)

        hasher.update(
            len(shape_bytes).to_bytes(8, "little")
        )

        hasher.update(shape_bytes)

        hasher.update(
            tensor.numpy().tobytes(order="C")
        )

    return hasher.hexdigest()


model_sha_now_629jr1 = _state_dict_sha256_629jr1(
    MODEL_FD003_629J
)

model_state_unchanged_629jr1 = bool(
    model_sha_now_629jr1
    ==
    FD003_INITIAL_MODEL_STATE_SHA256_629J
)

optimizer_state_entries_629jr1 = int(
    len(
        OPTIMIZER_FD003_629J
        .state_dict()["state"]
    )
)

optimizer_still_unstepped_629jr1 = bool(
    optimizer_state_entries_629jr1
    ==
    0
)

if not model_state_unchanged_629jr1:
    raise RuntimeError(
        "Persistent FD003 model state changed during DataLoader correction."
    )

if not optimizer_still_unstepped_629jr1:
    raise RuntimeError(
        "Optimizer state is no longer empty before training."
    )

# --------------------------------------------------------------------------------------
# 6) Correction contract
# --------------------------------------------------------------------------------------

FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_629JR1 = {
    "step":
        "6.29J-R1",

    "dataset":
        "FD003",

    "reason":
        "replace torch.from_numpy views of read-only frozen arrays with "
        "byte-identical writable C-contiguous copies before training",

    "prior_initialization_contract_sha256":
        FD003_INITIALIZATION_CONTRACT_SHA256_629J,

    "fit_x_content_exact":
        content_identity_629jr1["fit_x"],

    "fit_y_content_exact":
        content_identity_629jr1["fit_y"],

    "dev_x_content_exact":
        content_identity_629jr1["dev_x"],

    "dev_y_content_exact":
        content_identity_629jr1["dev_y"],

    "fit_x_writable":
        bool(FIT_X_WRITABLE_FD003_629JR1.flags.writeable),

    "fit_y_writable":
        bool(FIT_Y_WRITABLE_FD003_629JR1.flags.writeable),

    "dev_x_writable":
        bool(DEV_X_WRITABLE_FD003_629JR1.flags.writeable),

    "dev_y_writable":
        bool(DEV_Y_WRITABLE_FD003_629JR1.flags.writeable),

    "fit_loader_batch_count":
        int(len(FD003_FIT_LOADER_629JR1)),

    "dev_loader_batch_count":
        int(len(FD003_DEV_LOADER_629JR1)),

    "model_state_sha256":
        model_sha_now_629jr1,

    "model_state_unchanged":
        model_state_unchanged_629jr1,

    "optimizer_state_entries":
        optimizer_state_entries_629jr1,

    "optimizer_step_executed":
        False,

    "model_forward_executed":
        False,

    "model_training_performed":
        False,

    "dev_scoring_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

correction_payload_629jr1 = json.dumps(
    FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_629JR1,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_SHA256_629JR1 = (
    hashlib.sha256(
        correction_payload_629jr1
    ).hexdigest()
)

FD003_SAFE_DATALOADERS_READY_629JR1 = True

FD003_MODEL_FORWARD_EXECUTED_629JR1 = False
FD003_OPTIMIZER_STEP_EXECUTED_629JR1 = False
FD003_MODEL_TRAINING_PERFORMED_629JR1 = False
FD003_DEV_SCORING_PERFORMED_629JR1 = False
FD003_TEST_CONTENTS_READ_629JR1 = False
FD003_RUL_CONTENTS_READ_629JR1 = False

all_checks_passed_629jr1 = True

# --------------------------------------------------------------------------------------
# 7) Output
# --------------------------------------------------------------------------------------

print("=" * 126)

print(
    "STEP 6.29J-R1 — FD003 WRITABLE-BACKING "
    "DATALOADER SAFETY CORRECTION"
)

print("=" * 126)

print("Model reinitialized          :", False)
print("Model forward pass performed :", False)
print("Optimizer step performed     :", False)
print("Model training performed     :", False)
print("DEV scoring performed        :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- BYTE-IDENTITY CHECK ---")

print(
    "FIT X exact                  :",
    content_identity_629jr1["fit_x"],
)

print(
    "FIT y exact                  :",
    content_identity_629jr1["fit_y"],
)

print(
    "DEV X exact                  :",
    content_identity_629jr1["dev_x"],
)

print(
    "DEV y exact                  :",
    content_identity_629jr1["dev_y"],
)

print("\n--- WRITABLE BACKING CHECK ---")

print(
    "FIT X writable               :",
    FIT_X_WRITABLE_FD003_629JR1.flags.writeable,
)

print(
    "FIT y writable               :",
    FIT_Y_WRITABLE_FD003_629JR1.flags.writeable,
)

print(
    "DEV X writable               :",
    DEV_X_WRITABLE_FD003_629JR1.flags.writeable,
)

print(
    "DEV y writable               :",
    DEV_Y_WRITABLE_FD003_629JR1.flags.writeable,
)

print("\n--- SAFE DATALOADERS ---")

print(
    "FIT dataset size             :",
    len(FD003_FIT_DATASET_629JR1),
)

print(
    "DEV dataset size             :",
    len(FD003_DEV_DATASET_629JR1),
)

print(
    "FIT batch count              :",
    len(FD003_FIT_LOADER_629JR1),
)

print(
    "DEV batch count              :",
    len(FD003_DEV_LOADER_629JR1),
)

print("\n--- INITIALIZATION PRESERVATION ---")

print(
    "Initial model state unchanged:",
    model_state_unchanged_629jr1,
)

print(
    "Model-state SHA-256          :",
    model_sha_now_629jr1,
)

print(
    "Optimizer state entries      :",
    optimizer_state_entries_629jr1,
)

print(
    "\nCorrection-contract SHA-256  :",
    FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_SHA256_629JR1,
)

print(
    "\nPASS — the read-only NumPy backing warning is corrected before "
    "training. The data bytes, persistent model initialization, and "
    "optimizer state remain unchanged."
)

print(
    "Next: execute the single locked FD003 training run using "
    "FD003_FIT_LOADER_629JR1 and FD003_DEV_LOADER_629JR1."
)

In [ ]:
# STEP 6.29K
# FD003 single locked training run + best-epoch restore + trained-model fingerprint
# DEV monitoring ONLY / NO TEST ACCESS / NO OFFICIAL RUL ACCESS / NO RETUNING

import hashlib
import json
import math

import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629jr1", False)):
    raise RuntimeError(
        "STEP 6.29J-R1 must pass before the FD003 training run."
    )

required_629k = [
    "MODEL_FD003_629J",
    "OPTIMIZER_FD003_629J",
    "FD003_LOSS_FN_629J",
    "FD003_DEVICE_629J",
    "FD003_FIT_LOADER_629JR1",
    "FD003_DEV_LOADER_629JR1",
    "y_DEV_FD003_629F",
    "FD003_INITIAL_MODEL_STATE_SHA256_629J",
    "FD003_INITIALIZATION_CONTRACT_SHA256_629J",
    "FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_SHA256_629JR1",
    "FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I",
    "FD003_MAX_EPOCHS_629I",
    "FD003_EARLY_STOPPING_PATIENCE_629I",
    "FD003_EARLY_STOPPING_MIN_DELTA_629I",
    "FD003_RESTORE_BEST_WEIGHTS_629I",
    "FD003_TIE_BREAK_RULE_629I",
]

missing_629k = [
    name
    for name in required_629k
    if name not in globals()
]

if missing_629k:
    raise RuntimeError(
        "Missing required FD003 training object(s): "
        + ", ".join(missing_629k)
    )

if bool(globals().get("FD003_TRAINING_ATTEMPTED_629K", False)):
    raise RuntimeError(
        "STEP 6.29K training has already been attempted in this session. "
        "The locked single-run protocol forbids a second attempt."
    )

if bool(globals().get("FD003_MODEL_TRAINING_PERFORMED_629JR1", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates training already occurred."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629JR1", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629JR1", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

if str(FD003_TIE_BREAK_RULE_629I) != "earliest epoch":
    raise RuntimeError(
        "Unexpected early-stopping tie-break rule."
    )

if not bool(FD003_RESTORE_BEST_WEIGHTS_629I):
    raise RuntimeError(
        "Locked protocol requires restoring the best DEV-MAE epoch."
    )

# --------------------------------------------------------------------------------------
# 2) Fingerprint helpers + pre-training integrity checks
# --------------------------------------------------------------------------------------

def _state_dict_sha256_629k(model):
    hasher = hashlib.sha256()

    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes = key.encode("utf-8")
        dtype_bytes = str(tensor.dtype).encode("utf-8")
        shape_bytes = json.dumps(
            list(tensor.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(len(key_bytes).to_bytes(8, "little"))
        hasher.update(key_bytes)

        hasher.update(len(dtype_bytes).to_bytes(8, "little"))
        hasher.update(dtype_bytes)

        hasher.update(len(shape_bytes).to_bytes(8, "little"))
        hasher.update(shape_bytes)

        hasher.update(
            tensor.numpy().tobytes(order="C")
        )

    return hasher.hexdigest()


def _state_mapping_sha256_629k(state_mapping):
    hasher = hashlib.sha256()

    for key in sorted(state_mapping.keys()):
        tensor = (
            state_mapping[key]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes = key.encode("utf-8")
        dtype_bytes = str(tensor.dtype).encode("utf-8")
        shape_bytes = json.dumps(
            list(tensor.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(len(key_bytes).to_bytes(8, "little"))
        hasher.update(key_bytes)

        hasher.update(len(dtype_bytes).to_bytes(8, "little"))
        hasher.update(dtype_bytes)

        hasher.update(len(shape_bytes).to_bytes(8, "little"))
        hasher.update(shape_bytes)

        hasher.update(
            tensor.numpy().tobytes(order="C")
        )

    return hasher.hexdigest()


pretrain_model_sha_629k = _state_dict_sha256_629k(
    MODEL_FD003_629J
)

if pretrain_model_sha_629k != FD003_INITIAL_MODEL_STATE_SHA256_629J:
    raise RuntimeError(
        "Persistent FD003 model state changed before the locked training run."
    )

if len(OPTIMIZER_FD003_629J.state_dict()["state"]) != 0:
    raise RuntimeError(
        "Adam optimizer state is unexpectedly non-empty before training."
    )

if len(FD003_FIT_LOADER_629JR1.dataset) != 17114:
    raise RuntimeError(
        "FD003 FIT DataLoader size changed before training."
    )

if len(FD003_DEV_LOADER_629JR1.dataset) != 4706:
    raise RuntimeError(
        "FD003 DEV DataLoader size changed before training."
    )

# --------------------------------------------------------------------------------------
# 3) Evaluation helper
# --------------------------------------------------------------------------------------

@torch.inference_mode()
def _evaluate_dev_629k(model, loader, device):
    original_training_state_629k = bool(model.training)

    model.eval()

    abs_error_sum_629k = 0.0
    n_obs_629k = 0

    predictions_629k = []
    targets_629k = []

    for xb_629k, yb_629k in loader:
        xb_629k = xb_629k.to(
            device,
            non_blocking=True,
        )

        yb_629k = yb_629k.to(
            device,
            non_blocking=True,
        )

        pred_629k = model(
            xb_629k
        )

        if pred_629k.shape != yb_629k.shape:
            raise RuntimeError(
                "DEV prediction/target shape mismatch."
            )

        abs_error_sum_629k += float(
            torch.abs(
                pred_629k - yb_629k
            ).sum().item()
        )

        n_obs_629k += int(
            yb_629k.numel()
        )

        predictions_629k.append(
            pred_629k.detach()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

        targets_629k.append(
            yb_629k.detach()
            .cpu()
            .numpy()
            .astype(np.float64)
        )

    if n_obs_629k != len(loader.dataset):
        raise RuntimeError(
            "DEV evaluation observation-count mismatch."
        )

    prediction_vector_629k = np.concatenate(
        predictions_629k
    )

    target_vector_629k = np.concatenate(
        targets_629k
    )

    dev_mae_629k = float(
        abs_error_sum_629k
        /
        n_obs_629k
    )

    if original_training_state_629k:
        model.train()
    else:
        model.eval()

    return (
        dev_mae_629k,
        prediction_vector_629k,
        target_vector_629k,
    )

# --------------------------------------------------------------------------------------
# 4) Single locked training attempt
# --------------------------------------------------------------------------------------

FD003_TRAINING_ATTEMPTED_629K = True
FD003_OPTIMIZER_STEP_EXECUTED_629K = False

model_629k = MODEL_FD003_629J
optimizer_629k = OPTIMIZER_FD003_629J
loss_fn_629k = FD003_LOSS_FN_629J
device_629k = FD003_DEVICE_629J

best_dev_mae_629k = math.inf
best_epoch_629k = None
best_state_629k = None

epochs_without_improvement_629k = 0

history_rows_629k = []

for epoch_629k in range(
    1,
    int(FD003_MAX_EPOCHS_629I) + 1,
):
    model_629k.train()

    fit_squared_error_sum_629k = 0.0
    fit_n_629k = 0

    for xb_629k, yb_629k in FD003_FIT_LOADER_629JR1:
        xb_629k = xb_629k.to(
            device_629k,
            non_blocking=True,
        )

        yb_629k = yb_629k.to(
            device_629k,
            non_blocking=True,
        )

        optimizer_629k.zero_grad(
            set_to_none=True
        )

        pred_629k = model_629k(
            xb_629k
        )

        if pred_629k.shape != yb_629k.shape:
            raise RuntimeError(
                "FIT prediction/target shape mismatch."
            )

        loss_629k = loss_fn_629k(
            pred_629k,
            yb_629k,
        )

        if not torch.isfinite(loss_629k):
            raise RuntimeError(
                f"Non-finite FIT loss at epoch {epoch_629k}."
            )

        loss_629k.backward()

        optimizer_629k.step()

        FD003_OPTIMIZER_STEP_EXECUTED_629K = True

        batch_n_629k = int(
            yb_629k.numel()
        )

        fit_squared_error_sum_629k += float(
            loss_629k.detach().item()
        ) * batch_n_629k

        fit_n_629k += batch_n_629k

    if fit_n_629k != len(FD003_FIT_LOADER_629JR1.dataset):
        raise RuntimeError(
            "FIT epoch observation-count mismatch."
        )

    fit_mse_629k = float(
        fit_squared_error_sum_629k
        /
        fit_n_629k
    )

    dev_mae_629k, _, _ = _evaluate_dev_629k(
        model_629k,
        FD003_DEV_LOADER_629JR1,
        device_629k,
    )

    if not np.isfinite(fit_mse_629k):
        raise RuntimeError(
            f"Non-finite FIT MSE at epoch {epoch_629k}."
        )

    if not np.isfinite(dev_mae_629k):
        raise RuntimeError(
            f"Non-finite DEV MAE at epoch {epoch_629k}."
        )

    improved_629k = bool(
        dev_mae_629k
        <
        (
            best_dev_mae_629k
            -
            float(FD003_EARLY_STOPPING_MIN_DELTA_629I)
        )
    )

    if improved_629k:
        best_dev_mae_629k = float(
            dev_mae_629k
        )

        best_epoch_629k = int(
            epoch_629k
        )

        best_state_629k = {
            key_629k:
                value_629k.detach()
                .cpu()
                .clone()
            for key_629k, value_629k
            in model_629k.state_dict().items()
        }

        epochs_without_improvement_629k = 0

    else:
        epochs_without_improvement_629k += 1

    history_rows_629k.append(
        {
            "epoch":
                int(epoch_629k),

            "fit_mse":
                float(fit_mse_629k),

            "dev_mae":
                float(dev_mae_629k),

            "improved":
                bool(improved_629k),

            "epochs_without_improvement":
                int(epochs_without_improvement_629k),
        }
    )

    print(
        f"epoch={epoch_629k:03d}  "
        f"fit_mse={fit_mse_629k:.6f}  "
        f"dev_mae={dev_mae_629k:.6f}  "
        f"improved={improved_629k}  "
        f"bad_epochs={epochs_without_improvement_629k}"
    )

    if (
        epochs_without_improvement_629k
        >=
        int(FD003_EARLY_STOPPING_PATIENCE_629I)
    ):
        break

if not FD003_OPTIMIZER_STEP_EXECUTED_629K:
    raise RuntimeError(
        "Training loop completed without any optimizer step."
    )

if best_state_629k is None or best_epoch_629k is None:
    raise RuntimeError(
        "No best FD003 epoch was recorded."
    )

# --------------------------------------------------------------------------------------
# 5) Restore earliest best DEV-MAE epoch
# --------------------------------------------------------------------------------------

FD003_BEST_MODEL_STATE_SHA256_BEFORE_RESTORE_629K = (
    _state_mapping_sha256_629k(
        best_state_629k
    )
)

model_629k.load_state_dict(
    best_state_629k,
    strict=True,
)

model_629k.eval()

FD003_TRAINED_MODEL_STATE_SHA256_629K = (
    _state_dict_sha256_629k(
        model_629k
    )
)

if (
    FD003_TRAINED_MODEL_STATE_SHA256_629K
    !=
    FD003_BEST_MODEL_STATE_SHA256_BEFORE_RESTORE_629K
):
    raise RuntimeError(
        "Restored FD003 model does not exactly match the stored best state."
    )

# --------------------------------------------------------------------------------------
# 6) Final deterministic DEV prediction from the restored best model
# --------------------------------------------------------------------------------------

(
    final_dev_mae_629k,
    final_dev_prediction_629k,
    final_dev_target_629k,
) = _evaluate_dev_629k(
    model_629k,
    FD003_DEV_LOADER_629JR1,
    device_629k,
)

if not np.isclose(
    final_dev_mae_629k,
    best_dev_mae_629k,
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "Restored best-model DEV MAE does not match recorded best DEV MAE."
    )

if not np.array_equal(
    final_dev_target_629k.astype(np.float32),
    np.asarray(
        y_DEV_FD003_629F,
        dtype=np.float32,
    ),
):
    raise RuntimeError(
        "Final DEV target order does not match the frozen DEV target vector."
    )

FD003_DEV_DETERMINISTIC_PREDICTION_629K = np.array(
    final_dev_prediction_629k,
    dtype=np.float64,
    copy=True,
)

FD003_DEV_DETERMINISTIC_TARGET_629K = np.array(
    final_dev_target_629k,
    dtype=np.float64,
    copy=True,
)

FD003_DEV_DETERMINISTIC_ABS_ERROR_629K = np.abs(
    FD003_DEV_DETERMINISTIC_PREDICTION_629K
    -
    FD003_DEV_DETERMINISTIC_TARGET_629K
).astype(
    np.float64
)

for arr_629k in [
    FD003_DEV_DETERMINISTIC_PREDICTION_629K,
    FD003_DEV_DETERMINISTIC_TARGET_629K,
    FD003_DEV_DETERMINISTIC_ABS_ERROR_629K,
]:
    arr_629k.setflags(
        write=False
    )

# --------------------------------------------------------------------------------------
# 7) Training-history and prediction fingerprints
# --------------------------------------------------------------------------------------

FD003_TRAINING_HISTORY_629K = pd.DataFrame(
    history_rows_629k
)

history_payload_629k = FD003_TRAINING_HISTORY_629K.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

FD003_TRAINING_HISTORY_SHA256_629K = hashlib.sha256(
    history_payload_629k
).hexdigest()


def _raw_sha256_629k(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 training-result hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


FD003_DEV_PREDICTION_SHA256_629K = _raw_sha256_629k(
    FD003_DEV_DETERMINISTIC_PREDICTION_629K
)

FD003_DEV_TARGET_SHA256_629K = _raw_sha256_629k(
    FD003_DEV_DETERMINISTIC_TARGET_629K
)

FD003_DEV_ABS_ERROR_SHA256_629K = _raw_sha256_629k(
    FD003_DEV_DETERMINISTIC_ABS_ERROR_629K
)

# --------------------------------------------------------------------------------------
# 8) Final training-run contract
# --------------------------------------------------------------------------------------

epochs_run_629k = int(
    len(
        FD003_TRAINING_HISTORY_629K
    )
)

early_stopped_629k = bool(
    epochs_run_629k
    <
    int(FD003_MAX_EPOCHS_629I)
)

FD003_TRAINING_RUN_CONTRACT_629K = {
    "step":
        "6.29K",

    "dataset":
        "FD003",

    "training_protocol_contract_sha256":
        FD003_TRAINING_PROTOCOL_CONTRACT_SHA256_629I,

    "initialization_contract_sha256":
        FD003_INITIALIZATION_CONTRACT_SHA256_629J,

    "dataloader_safety_correction_contract_sha256":
        FD003_DATALOADER_SAFETY_CORRECTION_CONTRACT_SHA256_629JR1,

    "pretraining_model_state_sha256":
        pretrain_model_sha_629k,

    "single_training_attempt":
        True,

    "epochs_run":
        epochs_run_629k,

    "max_epochs":
        int(FD003_MAX_EPOCHS_629I),

    "early_stopped":
        early_stopped_629k,

    "early_stopping_patience":
        int(FD003_EARLY_STOPPING_PATIENCE_629I),

    "early_stopping_min_delta":
        float(FD003_EARLY_STOPPING_MIN_DELTA_629I),

    "monitor":
        "DEV_MAE",

    "tie_break_rule":
        "earliest epoch",

    "best_epoch":
        int(best_epoch_629k),

    "best_dev_mae":
        float(best_dev_mae_629k),

    "final_restored_dev_mae":
        float(final_dev_mae_629k),

    "restored_best_weights":
        True,

    "trained_model_state_sha256":
        FD003_TRAINED_MODEL_STATE_SHA256_629K,

    "training_history_sha256":
        FD003_TRAINING_HISTORY_SHA256_629K,

    "dev_prediction_sha256":
        FD003_DEV_PREDICTION_SHA256_629K,

    "dev_target_sha256":
        FD003_DEV_TARGET_SHA256_629K,

    "dev_abs_error_sha256":
        FD003_DEV_ABS_ERROR_SHA256_629K,

    "hyperparameter_search_performed":
        False,

    "second_training_run_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

training_run_payload_629k = json.dumps(
    FD003_TRAINING_RUN_CONTRACT_629K,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_TRAINING_RUN_CONTRACT_SHA256_629K = hashlib.sha256(
    training_run_payload_629k
).hexdigest()

FD003_MODEL_TRAINING_PERFORMED_629K = True
FD003_DEV_SCORING_PERFORMED_629K = True
FD003_BEST_WEIGHTS_RESTORED_629K = True
FD003_MODEL_LOCKED_AFTER_TRAINING_629K = True
FD003_TEST_CONTENTS_READ_629K = False
FD003_RUL_CONTENTS_READ_629K = False

all_checks_passed_629k = True

# --------------------------------------------------------------------------------------
# 9) Output
# --------------------------------------------------------------------------------------

print("\n" + "=" * 126)
print(
    "STEP 6.29K — FD003 SINGLE LOCKED TRAINING RUN COMPLETE"
)
print("=" * 126)

print("Training attempt count       :", 1)
print("Optimizer steps performed    :", True)
print("DEV monitoring performed     :", True)
print("Hyperparameter search        :", False)
print("Second training run          :", False)
print("TEST contents read           :", False)
print("Official RUL contents read   :", False)

print("\n--- EARLY-STOPPING RESULT ---")
print(
    "Epochs run                   :",
    epochs_run_629k,
)
print(
    "Early stopped                :",
    early_stopped_629k,
)
print(
    "Best epoch                   :",
    best_epoch_629k,
)
print(
    "Best DEV MAE                 :",
    best_dev_mae_629k,
)
print(
    "Restored-model DEV MAE       :",
    final_dev_mae_629k,
)
print(
    "Best weights restored exactly:",
    True,
)

print("\n--- FINAL TRAINED-MODEL EVIDENCE ---")
print(
    "Initial model-state SHA-256  :",
    FD003_INITIAL_MODEL_STATE_SHA256_629J,
)
print(
    "Best/trained state SHA-256   :",
    FD003_TRAINED_MODEL_STATE_SHA256_629K,
)
print(
    "Training-history SHA-256     :",
    FD003_TRAINING_HISTORY_SHA256_629K,
)
print(
    "DEV prediction SHA-256       :",
    FD003_DEV_PREDICTION_SHA256_629K,
)
print(
    "DEV target SHA-256           :",
    FD003_DEV_TARGET_SHA256_629K,
)
print(
    "DEV abs-error SHA-256        :",
    FD003_DEV_ABS_ERROR_SHA256_629K,
)
print(
    "Training-run contract SHA    :",
    FD003_TRAINING_RUN_CONTRACT_SHA256_629K,
)

print(
    "\nPASS — the single prospectively locked FD003 training run is complete. "
    "The earliest best DEV-MAE epoch has been restored and the trained model "
    "is now locked against retraining."
)

print(
    "Next: establish deterministic DEV baseline evidence and then the "
    "FD003 MC-dropout uncertainty contract using DEV only. "
    "TEST features and official RUL remain unopened."
)

In [ ]:
# STEP 6.29L
# FD003 deterministic clean-DEV baseline evidence lock
# VERIFY-AND-REUSE ONLY
# NO MODEL INFERENCE / NO TRAINING / NO MC DROPOUT / NO TEST OR RUL ACCESS

import hashlib
import json

import numpy as np
import pandas as pd
import torch

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629k", False)):
    raise RuntimeError(
        "STEP 6.29K must pass before locking the FD003 deterministic DEV baseline."
    )

required_629l = [
    "MODEL_FD003_629J",
    "FD003_TRAINED_MODEL_STATE_SHA256_629K",
    "FD003_TRAINING_RUN_CONTRACT_629K",
    "FD003_TRAINING_RUN_CONTRACT_SHA256_629K",
    "FD003_DEV_DETERMINISTIC_PREDICTION_629K",
    "FD003_DEV_DETERMINISTIC_TARGET_629K",
    "FD003_DEV_DETERMINISTIC_ABS_ERROR_629K",
    "FD003_DEV_PREDICTION_SHA256_629K",
    "FD003_DEV_TARGET_SHA256_629K",
    "FD003_DEV_ABS_ERROR_SHA256_629K",
    "FD003_DEV_WINDOW_KEYS_SHA256_629F",
]

missing_629l = [
    name
    for name in required_629l
    if name not in globals()
]

if missing_629l:
    raise RuntimeError(
        "Missing required FD003 deterministic-baseline object(s): "
        + ", ".join(missing_629l)
    )

if not bool(
    globals().get(
        "FD003_MODEL_LOCKED_AFTER_TRAINING_629K",
        False,
    )
):
    raise RuntimeError(
        "FD003 trained model is not marked locked after training."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629K", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629K", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Hash helpers
# --------------------------------------------------------------------------------------

def _raw_sha256_629l(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 baseline hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _state_dict_sha256_629l(model):
    hasher = hashlib.sha256()

    state_629l = model.state_dict()

    for key_629l in sorted(state_629l.keys()):
        tensor_629l = (
            state_629l[key_629l]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes_629l = key_629l.encode("utf-8")

        dtype_bytes_629l = str(
            tensor_629l.dtype
        ).encode("utf-8")

        shape_bytes_629l = json.dumps(
            list(tensor_629l.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(
            len(key_bytes_629l).to_bytes(
                8,
                "little",
            )
        )
        hasher.update(key_bytes_629l)

        hasher.update(
            len(dtype_bytes_629l).to_bytes(
                8,
                "little",
            )
        )
        hasher.update(dtype_bytes_629l)

        hasher.update(
            len(shape_bytes_629l).to_bytes(
                8,
                "little",
            )
        )
        hasher.update(shape_bytes_629l)

        hasher.update(
            tensor_629l.numpy().tobytes(
                order="C"
            )
        )

    return hasher.hexdigest()

# --------------------------------------------------------------------------------------
# 3) Verify trained model state has not changed after STEP 6.29K
# --------------------------------------------------------------------------------------

model_state_sha_now_629l = _state_dict_sha256_629l(
    MODEL_FD003_629J
)

model_state_exact_629l = bool(
    model_state_sha_now_629l
    ==
    FD003_TRAINED_MODEL_STATE_SHA256_629K
)

if not model_state_exact_629l:
    raise RuntimeError(
        "FD003 trained model state changed after STEP 6.29K."
    )

# --------------------------------------------------------------------------------------
# 4) Reuse existing deterministic DEV evidence only
# --------------------------------------------------------------------------------------

prediction_629l = np.asarray(
    FD003_DEV_DETERMINISTIC_PREDICTION_629K,
    dtype=np.float64,
)

target_629l = np.asarray(
    FD003_DEV_DETERMINISTIC_TARGET_629K,
    dtype=np.float64,
)

abs_error_629l = np.asarray(
    FD003_DEV_DETERMINISTIC_ABS_ERROR_629K,
    dtype=np.float64,
)

if prediction_629l.shape != (4706,):
    raise RuntimeError(
        "FD003 deterministic DEV prediction shape is not (4706,)."
    )

if target_629l.shape != (4706,):
    raise RuntimeError(
        "FD003 deterministic DEV target shape is not (4706,)."
    )

if abs_error_629l.shape != (4706,):
    raise RuntimeError(
        "FD003 deterministic DEV absolute-error shape is not (4706,)."
    )

if not np.isfinite(prediction_629l).all():
    raise RuntimeError(
        "FD003 deterministic DEV predictions contain non-finite values."
    )

if not np.isfinite(target_629l).all():
    raise RuntimeError(
        "FD003 deterministic DEV targets contain non-finite values."
    )

if not np.isfinite(abs_error_629l).all():
    raise RuntimeError(
        "FD003 deterministic DEV absolute errors contain non-finite values."
    )

# --------------------------------------------------------------------------------------
# 5) Independent prediction/error identity verification
# --------------------------------------------------------------------------------------

recomputed_abs_error_629l = np.abs(
    prediction_629l
    -
    target_629l
)

abs_error_exact_629l = bool(
    np.array_equal(
        recomputed_abs_error_629l,
        abs_error_629l,
    )
)

if not abs_error_exact_629l:
    raise RuntimeError(
        "Stored FD003 DEV absolute-error vector does not exactly equal |prediction-target|."
    )

prediction_sha_629l = _raw_sha256_629l(
    prediction_629l
)

target_sha_629l = _raw_sha256_629l(
    target_629l
)

abs_error_sha_629l = _raw_sha256_629l(
    abs_error_629l
)

prediction_hash_exact_629l = bool(
    prediction_sha_629l
    ==
    FD003_DEV_PREDICTION_SHA256_629K
)

target_hash_exact_629l = bool(
    target_sha_629l
    ==
    FD003_DEV_TARGET_SHA256_629K
)

abs_error_hash_exact_629l = bool(
    abs_error_sha_629l
    ==
    FD003_DEV_ABS_ERROR_SHA256_629K
)

if not prediction_hash_exact_629l:
    raise RuntimeError(
        "FD003 DEV deterministic prediction fingerprint changed."
    )

if not target_hash_exact_629l:
    raise RuntimeError(
        "FD003 DEV deterministic target fingerprint changed."
    )

if not abs_error_hash_exact_629l:
    raise RuntimeError(
        "FD003 DEV deterministic absolute-error fingerprint changed."
    )

# --------------------------------------------------------------------------------------
# 6) Clean DEV baseline metrics
#
# These are reporting/development metrics only.
# They do NOT authorize retraining or hyperparameter changes.
# --------------------------------------------------------------------------------------

FD003_CLEAN_DEV_MAE_629L = float(
    np.mean(
        abs_error_629l
    )
)

FD003_CLEAN_DEV_MEDIAN_ABS_ERROR_629L = float(
    np.median(
        abs_error_629l
    )
)

FD003_CLEAN_DEV_RMSE_629L = float(
    np.sqrt(
        np.mean(
            np.square(
                prediction_629l
                -
                target_629l
            )
        )
    )
)

training_monitor_mae_629l = float(
    FD003_TRAINING_RUN_CONTRACT_629K[
        "final_restored_dev_mae"
    ]
)

training_monitor_consistent_629l = bool(
    np.isclose(
        FD003_CLEAN_DEV_MAE_629L,
        training_monitor_mae_629l,
        rtol=0.0,
        atol=1e-5,
    )
)

if not training_monitor_consistent_629l:
    raise RuntimeError(
        "Clean DEV MAE is inconsistent with the restored-model DEV monitor."
    )

# --------------------------------------------------------------------------------------
# 7) Freeze canonical deterministic DEV evidence
# --------------------------------------------------------------------------------------

FD003_CLEAN_DEV_PREDICTION_629L = np.array(
    prediction_629l,
    dtype=np.float64,
    copy=True,
)

FD003_CLEAN_DEV_TARGET_629L = np.array(
    target_629l,
    dtype=np.float64,
    copy=True,
)

FD003_CLEAN_DEV_ABS_ERROR_629L = np.array(
    abs_error_629l,
    dtype=np.float64,
    copy=True,
)

for arr_629l in [
    FD003_CLEAN_DEV_PREDICTION_629L,
    FD003_CLEAN_DEV_TARGET_629L,
    FD003_CLEAN_DEV_ABS_ERROR_629L,
]:
    arr_629l.setflags(
        write=False
    )

# --------------------------------------------------------------------------------------
# 8) Baseline evidence contract
# --------------------------------------------------------------------------------------

FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L = {
    "step":
        "6.29L",

    "dataset":
        "FD003",

    "population":
        "all_4706_DEV_windows",

    "prediction_source":
        "reused restored-best-model deterministic DEV prediction from STEP 6.29K",

    "new_model_inference_performed":
        False,

    "model_state_sha256":
        model_state_sha_now_629l,

    "model_state_exact":
        model_state_exact_629l,

    "n_dev_windows":
        int(len(prediction_629l)),

    "clean_dev_mae":
        FD003_CLEAN_DEV_MAE_629L,

    "clean_dev_median_absolute_error":
        FD003_CLEAN_DEV_MEDIAN_ABS_ERROR_629L,

    "clean_dev_rmse":
        FD003_CLEAN_DEV_RMSE_629L,

    "training_monitor_mae":
        training_monitor_mae_629l,

    "training_monitor_consistent":
        training_monitor_consistent_629l,

    "prediction_sha256":
        prediction_sha_629l,

    "target_sha256":
        target_sha_629l,

    "absolute_error_sha256":
        abs_error_sha_629l,

    "dev_window_keys_sha256":
        FD003_DEV_WINDOW_KEYS_SHA256_629F,

    "absolute_error_exact":
        abs_error_exact_629l,

    "retraining_performed":
        False,

    "hyperparameter_retuning_performed":
        False,

    "mc_dropout_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

baseline_payload_629l = json.dumps(
    FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_SHA256_629L = (
    hashlib.sha256(
        baseline_payload_629l
    ).hexdigest()
)

FD003_DETERMINISTIC_DEV_BASELINE_LOCKED_629L = True
FD003_NEW_MODEL_INFERENCE_PERFORMED_629L = False
FD003_MC_DROPOUT_PERFORMED_629L = False
FD003_TEST_CONTENTS_READ_629L = False
FD003_RUL_CONTENTS_READ_629L = False

all_checks_passed_629l = True

# --------------------------------------------------------------------------------------
# 9) Output
# --------------------------------------------------------------------------------------

print("=" * 126)

print(
    "STEP 6.29L — FD003 DETERMINISTIC CLEAN-DEV "
    "BASELINE EVIDENCE LOCK"
)

print("=" * 126)

print("New model inference performed :", False)
print("Model retraining performed    :", False)
print("Hyperparameter retuning       :", False)
print("MC-dropout performed          :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- TRAINED MODEL INTEGRITY ---")

print(
    "Model state unchanged         :",
    model_state_exact_629l,
)

print(
    "Model-state SHA-256           :",
    model_state_sha_now_629l,
)

print("\n--- DETERMINISTIC DEV IDENTITY ---")

print(
    "DEV windows                   :",
    len(prediction_629l),
)

print(
    "Prediction hash exact         :",
    prediction_hash_exact_629l,
)

print(
    "Target hash exact             :",
    target_hash_exact_629l,
)

print(
    "Absolute-error hash exact     :",
    abs_error_hash_exact_629l,
)

print(
    "|prediction-target| exact     :",
    abs_error_exact_629l,
)

print("\n--- CLEAN DEV BASELINE ---")

print(
    "Clean DEV MAE                 :",
    FD003_CLEAN_DEV_MAE_629L,
)

print(
    "Clean DEV median abs. error   :",
    FD003_CLEAN_DEV_MEDIAN_ABS_ERROR_629L,
)

print(
    "Clean DEV RMSE                :",
    FD003_CLEAN_DEV_RMSE_629L,
)

print(
    "Training-monitor DEV MAE      :",
    training_monitor_mae_629l,
)

print(
    "Monitor/baseline consistent   :",
    training_monitor_consistent_629l,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Prediction SHA-256            :",
    prediction_sha_629l,
)

print(
    "Target SHA-256                :",
    target_sha_629l,
)

print(
    "Absolute-error SHA-256        :",
    abs_error_sha_629l,
)

print(
    "DEV window-keys SHA-256       :",
    FD003_DEV_WINDOW_KEYS_SHA256_629F,
)

print(
    "Baseline-contract SHA-256     :",
    FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_SHA256_629L,
)

print(
    "\nPASS — FD003 deterministic clean-DEV baseline evidence is "
    "verified, reused from the single locked training run, and frozen."
)

print(
    "Next: freeze and execute the canonical 50-pass FD003 DEV "
    "MC-dropout uncertainty procedure (seed=42, ddof=0), using DEV only."
)

In [ ]:
# STEP 6.29M
# FD003 clean-DEV canonical MC-dropout uncertainty execution + evidence lock
# DEV FEATURES ONLY / NO DEV LABEL USE / NO TRAINING / NO TEST OR OFFICIAL RUL ACCESS

import hashlib
import json
import random
import time

import numpy as np
import torch

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629l", False)):
    raise RuntimeError(
        "STEP 6.29L must pass before FD003 MC-dropout uncertainty execution."
    )

required_629m = [
    "MODEL_FD003_629J",
    "FD003_DEVICE_629J",
    "DEV_X_WRITABLE_FD003_629JR1",
    "FD003_TRAINED_MODEL_STATE_SHA256_629K",
    "FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_SHA256_629L",
    "FD003_CLEAN_DEV_PREDICTION_629L",
]

missing_629m = [
    name
    for name in required_629m
    if name not in globals()
]

if missing_629m:
    raise RuntimeError(
        "Missing required FD003 MC-dropout object(s): "
        + ", ".join(missing_629m)
    )

if bool(globals().get("FD003_MC_DROPOUT_EXECUTED_629M", False)):
    raise RuntimeError(
        "STEP 6.29M has already executed the canonical FD003 DEV MC-dropout run "
        "in this session. Do not rerun without a provenance reason."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629L", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629L", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Canonical project MC-dropout protocol
# --------------------------------------------------------------------------------------

FD003_MC_PASSES_629M = 50
FD003_MC_SEED_629M = 42
FD003_MC_BATCH_SIZE_629M = 512
FD003_MC_STD_DDOF_629M = 0

FD003_MC_PROTOCOL_629M = {
    "passes": FD003_MC_PASSES_629M,
    "seed": FD003_MC_SEED_629M,
    "batch_size": FD003_MC_BATCH_SIZE_629M,
    "mc_matrix_dtype": "float32",
    "mc_mean_dtype": "float64",
    "mc_std_dtype": "float64",
    "std_ddof": FD003_MC_STD_DDOF_629M,
    "dropout_activation": "model.train() during stochastic passes",
    "model_state_restore": "restore original train/eval mode after execution",
    "rng_scope": "seeded locally and notebook RNG states restored afterward",
    "population": "all_4706_clean_DEV_windows",
    "dev_labels_used": False,
    "test_used": False,
    "official_rul_used": False,
}

mc_protocol_payload_629m = json.dumps(
    FD003_MC_PROTOCOL_629M,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_MC_PROTOCOL_SHA256_629M = hashlib.sha256(
    mc_protocol_payload_629m
).hexdigest()

# --------------------------------------------------------------------------------------
# 3) Fingerprint helpers
# --------------------------------------------------------------------------------------

def _raw_sha256_629m(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during FD003 MC hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _state_dict_sha256_629m(model):
    hasher = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes = key.encode("utf-8")
        dtype_bytes = str(tensor.dtype).encode("utf-8")
        shape_bytes = json.dumps(
            list(tensor.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher.update(len(key_bytes).to_bytes(8, "little"))
        hasher.update(key_bytes)

        hasher.update(len(dtype_bytes).to_bytes(8, "little"))
        hasher.update(dtype_bytes)

        hasher.update(len(shape_bytes).to_bytes(8, "little"))
        hasher.update(shape_bytes)

        hasher.update(
            tensor.numpy().tobytes(order="C")
        )

    return hasher.hexdigest()

# --------------------------------------------------------------------------------------
# 4) Pre-execution integrity
# --------------------------------------------------------------------------------------

model_629m = MODEL_FD003_629J
device_629m = FD003_DEVICE_629J

model_sha_before_629m = _state_dict_sha256_629m(
    model_629m
)

if (
    model_sha_before_629m
    !=
    FD003_TRAINED_MODEL_STATE_SHA256_629K
):
    raise RuntimeError(
        "FD003 trained model state changed before MC-dropout execution."
    )

X_dev_629m = np.asarray(
    DEV_X_WRITABLE_FD003_629JR1,
    dtype=np.float32,
)

if X_dev_629m.shape != (4706, 30, 18):
    raise RuntimeError(
        "FD003 clean DEV feature tensor shape changed before MC execution."
    )

if not np.isfinite(X_dev_629m).all():
    raise RuntimeError(
        "FD003 clean DEV feature tensor contains non-finite values."
    )

# --------------------------------------------------------------------------------------
# 5) Canonical MC-dropout helper
#
# DEV labels are never passed to this helper.
# The deterministic 6.29L prediction remains the canonical point prediction.
# --------------------------------------------------------------------------------------

def mc_dropout_predict_fd003_629m(
    X,
    model,
    device,
    passes=50,
    batch_size=512,
    seed=42,
):
    X = np.asarray(
        X,
        dtype=np.float32,
    )

    n = int(len(X))

    if n <= 0:
        raise RuntimeError(
            "MC-dropout received an empty input tensor."
        )

    all_predictions = np.empty(
        (
            int(passes),
            n,
        ),
        dtype=np.float32,
    )

    python_rng_state = random.getstate()
    numpy_rng_state = np.random.get_state()
    torch_cpu_rng_state = torch.random.get_rng_state()

    torch_cuda_rng_states = None
    if torch.cuda.is_available():
        torch_cuda_rng_states = torch.cuda.get_rng_state_all()

    original_training_state = bool(
        model.training
    )

    start_time = time.time()

    try:
        random.seed(int(seed))
        np.random.seed(int(seed))
        torch.manual_seed(int(seed))

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(
                int(seed)
            )

        model.train()

        with torch.no_grad():
            for pass_idx in range(
                int(passes)
            ):
                for start_idx in range(
                    0,
                    n,
                    int(batch_size),
                ):
                    end_idx = min(
                        start_idx
                        +
                        int(batch_size),
                        n,
                    )

                    xb = torch.as_tensor(
                        X[start_idx:end_idx],
                        dtype=torch.float32,
                        device=device,
                    )

                    pred = (
                        model(xb)
                        .reshape(-1)
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(
                            np.float32,
                            copy=False,
                        )
                    )

                    if len(pred) != (
                        end_idx - start_idx
                    ):
                        raise RuntimeError(
                            "FD003 MC-dropout batch prediction count mismatch."
                        )

                    all_predictions[
                        pass_idx,
                        start_idx:end_idx
                    ] = pred

    finally:
        if original_training_state:
            model.train()
        else:
            model.eval()

        random.setstate(
            python_rng_state
        )

        np.random.set_state(
            numpy_rng_state
        )

        torch.random.set_rng_state(
            torch_cpu_rng_state
        )

        if torch_cuda_rng_states is not None:
            torch.cuda.set_rng_state_all(
                torch_cuda_rng_states
            )

    runtime_seconds = float(
        time.time() - start_time
    )

    mc_mean = (
        all_predictions
        .mean(axis=0)
        .astype(np.float64)
    )

    mc_std = (
        all_predictions
        .std(
            axis=0,
            ddof=0,
        )
        .astype(np.float64)
    )

    return (
        all_predictions,
        mc_mean,
        mc_std,
        runtime_seconds,
    )

# --------------------------------------------------------------------------------------
# 6) Execute exactly one canonical clean-DEV MC-dropout run
# --------------------------------------------------------------------------------------

FD003_MC_DROPOUT_EXECUTED_629M = True

(
    FD003_DEV_MC_MATRIX_629M,
    FD003_DEV_MC_MEAN_629M,
    FD003_DEV_MC_STD_629M,
    FD003_DEV_MC_RUNTIME_SECONDS_629M,
) = mc_dropout_predict_fd003_629m(
    X_dev_629m,
    model_629m,
    device_629m,
    passes=FD003_MC_PASSES_629M,
    batch_size=FD003_MC_BATCH_SIZE_629M,
    seed=FD003_MC_SEED_629M,
)

# --------------------------------------------------------------------------------------
# 7) Post-execution integrity
# --------------------------------------------------------------------------------------

if FD003_DEV_MC_MATRIX_629M.shape != (50, 4706):
    raise RuntimeError(
        "FD003 DEV MC matrix shape mismatch."
    )

if FD003_DEV_MC_MATRIX_629M.dtype != np.float32:
    raise RuntimeError(
        "FD003 DEV MC matrix dtype is not float32."
    )

if FD003_DEV_MC_MEAN_629M.shape != (4706,):
    raise RuntimeError(
        "FD003 DEV MC mean shape mismatch."
    )

if FD003_DEV_MC_STD_629M.shape != (4706,):
    raise RuntimeError(
        "FD003 DEV MC STD shape mismatch."
    )

if not np.isfinite(
    FD003_DEV_MC_MATRIX_629M
).all():
    raise RuntimeError(
        "FD003 DEV MC matrix contains non-finite values."
    )

if not np.isfinite(
    FD003_DEV_MC_MEAN_629M
).all():
    raise RuntimeError(
        "FD003 DEV MC mean contains non-finite values."
    )

if not np.isfinite(
    FD003_DEV_MC_STD_629M
).all():
    raise RuntimeError(
        "FD003 DEV MC STD contains non-finite values."
    )

if bool(
    (
        FD003_DEV_MC_STD_629M
        <
        0.0
    ).any()
):
    raise RuntimeError(
        "FD003 DEV MC STD contains a negative value."
    )

mc_mean_rederived_629m = (
    FD003_DEV_MC_MATRIX_629M
    .mean(axis=0)
    .astype(np.float64)
)

mc_std_rederived_629m = (
    FD003_DEV_MC_MATRIX_629M
    .std(
        axis=0,
        ddof=0,
    )
    .astype(np.float64)
)

mc_mean_exact_629m = bool(
    np.array_equal(
        FD003_DEV_MC_MEAN_629M,
        mc_mean_rederived_629m,
    )
)

mc_std_exact_629m = bool(
    np.array_equal(
        FD003_DEV_MC_STD_629M,
        mc_std_rederived_629m,
    )
)

if not mc_mean_exact_629m:
    raise RuntimeError(
        "FD003 MC mean is not exactly rederived from the MC matrix."
    )

if not mc_std_exact_629m:
    raise RuntimeError(
        "FD003 MC STD is not exactly rederived with ddof=0."
    )

model_sha_after_629m = _state_dict_sha256_629m(
    model_629m
)

model_state_unchanged_629m = bool(
    model_sha_after_629m
    ==
    model_sha_before_629m
)

if not model_state_unchanged_629m:
    raise RuntimeError(
        "FD003 trained model weights changed during MC-dropout execution."
    )

model_eval_restored_629m = bool(
    model_629m.training
    is False
)

if not model_eval_restored_629m:
    raise RuntimeError(
        "FD003 trained model was not restored to eval mode after MC-dropout."
    )

# --------------------------------------------------------------------------------------
# 8) Freeze MC artifacts read-only
# --------------------------------------------------------------------------------------

for arr_629m in [
    FD003_DEV_MC_MATRIX_629M,
    FD003_DEV_MC_MEAN_629M,
    FD003_DEV_MC_STD_629M,
]:
    arr_629m.setflags(
        write=False
    )

# --------------------------------------------------------------------------------------
# 9) Evidence fingerprints
# --------------------------------------------------------------------------------------

FD003_DEV_MC_MATRIX_SHA256_629M = _raw_sha256_629m(
    FD003_DEV_MC_MATRIX_629M
)

FD003_DEV_MC_MEAN_SHA256_629M = _raw_sha256_629m(
    FD003_DEV_MC_MEAN_629M
)

FD003_DEV_MC_STD_SHA256_629M = _raw_sha256_629m(
    FD003_DEV_MC_STD_629M
)

det_prediction_629m = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
)

FD003_MC_MEAN_VS_DETERMINISTIC_MAE_629M = float(
    np.mean(
        np.abs(
            FD003_DEV_MC_MEAN_629M
            -
            det_prediction_629m
        )
    )
)

# --------------------------------------------------------------------------------------
# 10) MC evidence contract
# --------------------------------------------------------------------------------------

FD003_MC_DEV_CONTRACT_629M = {
    "step":
        "6.29M",

    "dataset":
        "FD003",

    "population":
        "all_4706_clean_DEV_windows",

    "protocol_sha256":
        FD003_MC_PROTOCOL_SHA256_629M,

    "deterministic_baseline_contract_sha256":
        FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_SHA256_629L,

    "trained_model_state_sha256":
        model_sha_after_629m,

    "passes":
        FD003_MC_PASSES_629M,

    "seed":
        FD003_MC_SEED_629M,

    "batch_size":
        FD003_MC_BATCH_SIZE_629M,

    "std_ddof":
        FD003_MC_STD_DDOF_629M,

    "mc_matrix_shape":
        list(
            FD003_DEV_MC_MATRIX_629M.shape
        ),

    "mc_matrix_dtype":
        str(
            FD003_DEV_MC_MATRIX_629M.dtype
        ),

    "mc_mean_dtype":
        str(
            FD003_DEV_MC_MEAN_629M.dtype
        ),

    "mc_std_dtype":
        str(
            FD003_DEV_MC_STD_629M.dtype
        ),

    "mc_mean_exactly_rederived":
        mc_mean_exact_629m,

    "mc_std_exactly_rederived":
        mc_std_exact_629m,

    "model_state_unchanged":
        model_state_unchanged_629m,

    "model_eval_restored":
        model_eval_restored_629m,

    "mc_matrix_sha256":
        FD003_DEV_MC_MATRIX_SHA256_629M,

    "mc_mean_sha256":
        FD003_DEV_MC_MEAN_SHA256_629M,

    "mc_std_sha256":
        FD003_DEV_MC_STD_SHA256_629M,

    "canonical_point_prediction":
        "FD003_CLEAN_DEV_PREDICTION_629L",

    "mc_mean_replaces_point_prediction":
        False,

    "dev_labels_used_for_mc_execution":
        False,

    "model_retraining_performed":
        False,

    "hyperparameter_retuning_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

mc_contract_payload_629m = json.dumps(
    FD003_MC_DEV_CONTRACT_629M,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_MC_DEV_CONTRACT_SHA256_629M = hashlib.sha256(
    mc_contract_payload_629m
).hexdigest()

FD003_MC_DEV_LOCKED_629M = True
FD003_MODEL_RETRAINED_629M = False
FD003_DEV_LABELS_USED_FOR_MC_629M = False
FD003_TEST_CONTENTS_READ_629M = False
FD003_RUL_CONTENTS_READ_629M = False

all_checks_passed_629m = True

# --------------------------------------------------------------------------------------
# 11) Output
# --------------------------------------------------------------------------------------

print("=" * 126)

print(
    "STEP 6.29M — FD003 CLEAN-DEV CANONICAL "
    "MC-DROPOUT UNCERTAINTY EXECUTION + EVIDENCE LOCK"
)

print("=" * 126)

print("Model retraining performed    :", False)
print("DEV labels used for MC        :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- CANONICAL MC PROTOCOL ---")

print(
    "MC passes                     :",
    FD003_MC_PASSES_629M,
)

print(
    "MC seed                       :",
    FD003_MC_SEED_629M,
)

print(
    "MC batch size                 :",
    FD003_MC_BATCH_SIZE_629M,
)

print(
    "MC STD ddof                   :",
    FD003_MC_STD_DDOF_629M,
)

print(
    "MC protocol SHA-256           :",
    FD003_MC_PROTOCOL_SHA256_629M,
)

print("\n--- MC ARTIFACT CONTRACT ---")

print(
    "MC matrix shape               :",
    FD003_DEV_MC_MATRIX_629M.shape,
)

print(
    "MC matrix dtype               :",
    FD003_DEV_MC_MATRIX_629M.dtype,
)

print(
    "MC mean dtype                 :",
    FD003_DEV_MC_MEAN_629M.dtype,
)

print(
    "MC STD dtype                  :",
    FD003_DEV_MC_STD_629M.dtype,
)

print(
    "MC mean exactly rederived     :",
    mc_mean_exact_629m,
)

print(
    "MC STD exactly rederived      :",
    mc_std_exact_629m,
)

print(
    "MC runtime seconds            :",
    FD003_DEV_MC_RUNTIME_SECONDS_629M,
)

print("\n--- MODEL INTEGRITY ---")

print(
    "Model state unchanged         :",
    model_state_unchanged_629m,
)

print(
    "Model eval mode restored      :",
    model_eval_restored_629m,
)

print(
    "Trained model SHA-256         :",
    model_sha_after_629m,
)

print("\n--- UNCERTAINTY SUMMARY ---")

print(
    "MC STD min                    :",
    float(
        FD003_DEV_MC_STD_629M.min()
    ),
)

print(
    "MC STD median                 :",
    float(
        np.median(
            FD003_DEV_MC_STD_629M
        )
    ),
)

print(
    "MC STD mean                   :",
    float(
        FD003_DEV_MC_STD_629M.mean()
    ),
)

print(
    "MC STD max                    :",
    float(
        FD003_DEV_MC_STD_629M.max()
    ),
)

print(
    "Mean |MC mean - deterministic|:",
    FD003_MC_MEAN_VS_DETERMINISTIC_MAE_629M,
)

print("\n--- MC EVIDENCE FINGERPRINTS ---")

print(
    "MC matrix SHA-256             :",
    FD003_DEV_MC_MATRIX_SHA256_629M,
)

print(
    "MC mean SHA-256               :",
    FD003_DEV_MC_MEAN_SHA256_629M,
)

print(
    "MC STD SHA-256                :",
    FD003_DEV_MC_STD_SHA256_629M,
)

print(
    "MC DEV contract SHA-256       :",
    FD003_MC_DEV_CONTRACT_SHA256_629M,
)

print(
    "\nPASS — the canonical 50-pass FD003 clean-DEV MC-dropout "
    "uncertainty evidence is executed once, fingerprinted, and locked."
)

print(
    "The canonical RUL point prediction remains the frozen deterministic "
    "STEP 6.29L prediction; MC STD is the uncertainty signal."
)

print(
    "Next: calibrate/evaluate the label-free uncertainty reliability risk "
    "on DEV and establish the clean selective-prediction evidence before "
    "controlled corruption experiments."
)

In [ ]:
# STEP 6.29N
# FD003 historical uncertainty-risk calibration provenance audit
# READ-ONLY / NO MODEL EXECUTION / NO MC EXECUTION / NO DEV SCORING
# NO TEST ACCESS / NO OFFICIAL RUL ACCESS / NO PROJECT-STATE MODIFICATION

import dis
import inspect
import hashlib
import json
import pandas as pd

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629m", False)):
    raise RuntimeError(
        "STEP 6.29M must pass before uncertainty-risk calibration provenance recovery."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629M", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates TEST contents were read."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629M", True)):
    raise RuntimeError(
        "FD003 state unexpectedly indicates official RUL contents were read."
    )

# --------------------------------------------------------------------------------------
# 2) Strict historical-name inventory
#
# We inspect NAMES / TYPES / callable structure only.
# No historical array/table values are opened in this step.
# --------------------------------------------------------------------------------------

historical_markers_629n = (
    "628",
    "fd001",
    "fd002",
)

calibration_markers_629n = (
    "uncert",
    "risk",
    "calib",
    "rank",
    "percent",
    "ecdf",
    "cdf",
)

forbidden_name_markers_629n = (
    "test",
    "rul",
    "label",
    "error",
    "mae",
    "score",
    "scoring",
    "target",
)

candidate_names_629n = []

for name_629n in sorted(globals().keys()):
    lower_629n = name_629n.lower()

    historical_match_629n = any(
        marker_629n in lower_629n
        for marker_629n in historical_markers_629n
    )

    calibration_match_629n = any(
        marker_629n in lower_629n
        for marker_629n in calibration_markers_629n
    )

    forbidden_match_629n = any(
        marker_629n in lower_629n
        for marker_629n in forbidden_name_markers_629n
    )

    if (
        historical_match_629n
        and calibration_match_629n
        and not forbidden_match_629n
    ):
        candidate_names_629n.append(
            name_629n
        )

# --------------------------------------------------------------------------------------
# 3) Type/signature/key inventory only
# --------------------------------------------------------------------------------------

inventory_rows_629n = []
callable_names_629n = []
dict_names_629n = []

for name_629n in candidate_names_629n:
    obj_629n = globals()[name_629n]

    row_629n = {
        "name": name_629n,
        "type": type(obj_629n).__name__,
        "callable": bool(callable(obj_629n)),
        "signature": None,
        "dict_key_count": None,
        "dict_keys": None,
    }

    if callable(obj_629n):
        callable_names_629n.append(
            name_629n
        )

        try:
            row_629n["signature"] = str(
                inspect.signature(
                    obj_629n
                )
            )
        except (TypeError, ValueError):
            row_629n["signature"] = "<unavailable>"

    elif isinstance(obj_629n, dict):
        dict_names_629n.append(
            name_629n
        )

        safe_keys_629n = sorted(
            str(key_629n)
            for key_629n in obj_629n.keys()
            if not any(
                marker_629n in str(key_629n).lower()
                for marker_629n in forbidden_name_markers_629n
            )
        )

        row_629n["dict_key_count"] = int(
            len(obj_629n)
        )

        row_629n["dict_keys"] = safe_keys_629n

    inventory_rows_629n.append(
        row_629n
    )

FD003_HISTORICAL_RISK_PROVENANCE_INVENTORY_629N = pd.DataFrame(
    inventory_rows_629n
)

# --------------------------------------------------------------------------------------
# 4) Recover callable implementations WITHOUT executing them
#
# Prefer source. If source is unavailable, retain bytecode/disassembly metadata.
# --------------------------------------------------------------------------------------

callable_records_629n = {}

for name_629n in callable_names_629n:
    fn_629n = globals()[name_629n]

    record_629n = {
        "signature": None,
        "source_available": False,
        "source": None,
        "source_sha256": None,
        "bytecode_sha256": None,
        "names": None,
        "varnames": None,
        "constants_repr": None,
        "disassembly": None,
    }

    try:
        record_629n["signature"] = str(
            inspect.signature(
                fn_629n
            )
        )
    except (TypeError, ValueError):
        record_629n["signature"] = "<unavailable>"

    try:
        source_629n = inspect.getsource(
            fn_629n
        )

        record_629n["source_available"] = True
        record_629n["source"] = source_629n
        record_629n["source_sha256"] = hashlib.sha256(
            source_629n.encode("utf-8")
        ).hexdigest()

    except (OSError, TypeError):
        pass

    if hasattr(fn_629n, "__code__"):
        code_629n = fn_629n.__code__

        record_629n["bytecode_sha256"] = hashlib.sha256(
            code_629n.co_code
        ).hexdigest()

        record_629n["names"] = list(
            code_629n.co_names
        )

        record_629n["varnames"] = list(
            code_629n.co_varnames
        )

        record_629n["constants_repr"] = [
            repr(value_629n)
            for value_629n in code_629n.co_consts
        ]

        record_629n["disassembly"] = [
            {
                "offset": int(inst_629n.offset),
                "opname": inst_629n.opname,
                "argrepr": inst_629n.argrepr,
            }
            for inst_629n in dis.get_instructions(
                fn_629n
            )
        ]

    callable_records_629n[
        name_629n
    ] = record_629n

# --------------------------------------------------------------------------------------
# 5) Provenance audit fingerprint
# --------------------------------------------------------------------------------------

fingerprint_payload_629n = json.dumps(
    {
        "candidate_names":
            candidate_names_629n,

        "callables":
            {
                name_629n: {
                    "signature":
                        record_629n["signature"],

                    "source_available":
                        record_629n["source_available"],

                    "source_sha256":
                        record_629n["source_sha256"],

                    "bytecode_sha256":
                        record_629n["bytecode_sha256"],

                    "names":
                        record_629n["names"],

                    "varnames":
                        record_629n["varnames"],

                    "constants_repr":
                        record_629n["constants_repr"],
                }
                for name_629n, record_629n
                in callable_records_629n.items()
            },

        "dict_names":
            dict_names_629n,

        "new_fd003_risk_created":
            False,

        "dev_scoring_performed":
            False,

        "model_execution_performed":
            False,

        "mc_execution_performed":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_HISTORICAL_RISK_PROVENANCE_AUDIT_SHA256_629N = hashlib.sha256(
    fingerprint_payload_629n
).hexdigest()

FD003_NEW_RISK_CREATED_629N = False
FD003_DEV_SCORING_PERFORMED_629N = False
FD003_MODEL_EXECUTION_PERFORMED_629N = False
FD003_MC_EXECUTION_PERFORMED_629N = False
FD003_TEST_CONTENTS_READ_629N = False
FD003_RUL_CONTENTS_READ_629N = False

all_checks_passed_629n = True

# --------------------------------------------------------------------------------------
# 6) Output
# --------------------------------------------------------------------------------------

print("=" * 126)
print(
    "STEP 6.29N — FD003 HISTORICAL UNCERTAINTY-RISK "
    "CALIBRATION PROVENANCE AUDIT"
)
print("=" * 126)

print("New FD003 risk created        :", False)
print("DEV scoring performed         :", False)
print("Model execution performed     :", False)
print("MC-dropout execution performed:", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- STRICT HISTORICAL NAME INVENTORY ---")

if len(FD003_HISTORICAL_RISK_PROVENANCE_INVENTORY_629N) == 0:
    print("<no matching historical calibration/risk objects found>")
else:
    print(
        FD003_HISTORICAL_RISK_PROVENANCE_INVENTORY_629N.to_string(
            index=False
        )
    )

print("\n--- CALLABLE IMPLEMENTATION RECOVERY ---")

if not callable_records_629n:
    print("<no matching historical callables found>")
else:
    for name_629n, record_629n in callable_records_629n.items():
        print("\n" + "-" * 110)
        print(name_629n)
        print("-" * 110)

        print(
            "signature        :",
            record_629n["signature"],
        )

        print(
            "source available :",
            record_629n["source_available"],
        )

        print(
            "source SHA-256   :",
            record_629n["source_sha256"],
        )

        print(
            "bytecode SHA-256 :",
            record_629n["bytecode_sha256"],
        )

        if record_629n["source_available"]:
            print("\nSOURCE:")
            print(
                record_629n["source"]
            )

        elif record_629n["disassembly"] is not None:
            print("\nDISASSEMBLY:")

            for inst_629n in record_629n["disassembly"]:
                print(
                    f"{inst_629n['offset']:4d}  "
                    f"{inst_629n['opname']:28s} "
                    f"{inst_629n['argrepr']}"
                )

print(
    "\nHistorical-risk provenance audit SHA-256 :",
    FD003_HISTORICAL_RISK_PROVENANCE_AUDIT_SHA256_629N,
)

print(
    "\nPASS — historical uncertainty-risk calibration provenance "
    "was inspected structurally only. No FD003 risk or threshold "
    "has been created yet."
)

print(
    "Next action depends on this output: if an exact historical "
    "uncertainty-risk calibration rule is recovered, verify and reuse "
    "that methodology for FD003 using FD003-specific calibration data; "
    "otherwise recover the mapping from whitelisted historical "
    "pre-label artifacts before defining a new rule."
)

In [ ]:
# STEP 6.29O
# FD003 — corrected historical uncertainty-risk reference-construction recovery
#
# READ ONLY:
#   - no model execution
#   - no MC execution
#   - no DEV scoring / labels
#   - no TEST access
#   - no official RUL access
#   - no FD003 risk construction
#   - no threshold selection

import inspect
import hashlib
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629n", False)):
    raise RuntimeError(
        "STEP 6.29N must pass before STEP 6.29O."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629N", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29N."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629N", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29N."
    )


# ======================================================================================
# 2) Exact whitelist
# ======================================================================================

required_names_629o = [
    "FD002_UNCERTAINTY_RECIPE_628FBR1",
    "FD002_UNCERTAINTY_ONLY_CONTRACT_628HC1D",

    "apply_phase_adjustment_628fbr1",

    "apply_g1_right_ecdf_628fc",
    "apply_fd002_g2_weighted_midrank_628hc0r2",

    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",

    "G1_REFERENCE_FD002_628FC",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
]

missing_629o = [
    name
    for name in required_names_629o
    if name not in globals()
]

if missing_629o:
    raise RuntimeError(
        "Missing required historical object(s): "
        + ", ".join(missing_629o)
    )


# ======================================================================================
# 3) Safe metadata representation
#
# Primitive metadata can be printed.
# Arrays are represented ONLY by structural/fingerprint descriptors.
# ======================================================================================

def sha_array_629o(values):

    arr = np.ascontiguousarray(
        np.asarray(values)
    )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def safe_metadata_629o(value):

    if value is None or isinstance(
        value,
        (
            str,
            bool,
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        if isinstance(value, (np.integer,)):
            return int(value)

        if isinstance(value, (np.floating,)):
            return float(value)

        return value


    if isinstance(value, np.ndarray):

        arr = np.asarray(value)

        descriptor = {
            "kind": "ndarray",
            "shape": tuple(
                int(x)
                for x in arr.shape
            ),
            "dtype": str(arr.dtype),
            "finite": (
                bool(np.isfinite(arr).all())
                if np.issubdtype(
                    arr.dtype,
                    np.number,
                )
                else None
            ),
            "sha256": sha_array_629o(
                arr
            ),
        }

        return descriptor


    if isinstance(value, dict):

        return {
            str(k): safe_metadata_629o(v)
            for k, v in value.items()
        }


    if isinstance(value, (list, tuple)):

        # Small primitive metadata lists may be shown directly.
        if (
            len(value) <= 50
            and all(
                (
                    item is None
                    or isinstance(
                        item,
                        (
                            str,
                            bool,
                            int,
                            float,
                            np.integer,
                            np.floating,
                        ),
                    )
                )
                for item in value
            )
        ):
            return [
                safe_metadata_629o(item)
                for item in value
            ]

        return {
            "kind": type(value).__name__,
            "length": int(
                len(value)
            ),
        }


    return {
        "kind": type(value).__name__
    }


# ======================================================================================
# 4) Corrected historical recipe metadata
# ======================================================================================

recipe_fbr1_629o = safe_metadata_629o(
    FD002_UNCERTAINTY_RECIPE_628FBR1
)

uncertainty_contract_629o = safe_metadata_629o(
    FD002_UNCERTAINTY_ONLY_CONTRACT_628HC1D
)


# ======================================================================================
# 5) Exact callable-source recovery
# ======================================================================================

helper_names_629o = [
    "apply_phase_adjustment_628fbr1",
    "apply_g1_right_ecdf_628fc",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
]

helper_records_629o = {}

for name_629o in helper_names_629o:

    fn_629o = globals()[
        name_629o
    ]

    try:
        signature_629o = str(
            inspect.signature(
                fn_629o
            )
        )
    except (TypeError, ValueError):
        signature_629o = "<unavailable>"

    try:
        source_629o = inspect.getsource(
            fn_629o
        )
        source_available_629o = True
        source_sha_629o = hashlib.sha256(
            source_629o.encode(
                "utf-8"
            )
        ).hexdigest()

    except (OSError, TypeError):
        source_629o = None
        source_available_629o = False
        source_sha_629o = None

    code_names_629o = None

    if hasattr(
        fn_629o,
        "__code__",
    ):
        code_names_629o = list(
            fn_629o.__code__.co_names
        )

    helper_records_629o[
        name_629o
    ] = {
        "signature":
            signature_629o,

        "source_available":
            source_available_629o,

        "source_sha256":
            source_sha_629o,

        "source":
            source_629o,

        "referenced_global_names":
            code_names_629o,
    }


# ======================================================================================
# 6) G1 reference audit
# ======================================================================================

g1_ref_629o = np.asarray(
    G1_REFERENCE_FD002_628FC,
    dtype=np.float64,
).reshape(-1)

g1_audit_629o = {
    "shape":
        tuple(
            int(x)
            for x in g1_ref_629o.shape
        ),

    "dtype":
        str(
            g1_ref_629o.dtype
        ),

    "finite":
        bool(
            np.isfinite(
                g1_ref_629o
            ).all()
        ),

    "sorted_non_decreasing":
        bool(
            np.all(
                np.diff(
                    g1_ref_629o
                )
                >= 0
            )
        ),

    "unique_count":
        int(
            len(
                np.unique(
                    g1_ref_629o
                )
            )
        ),

    "minimum":
        float(
            g1_ref_629o.min()
        ),

    "maximum":
        float(
            g1_ref_629o.max()
        ),

    "sha256":
        sha_array_629o(
            g1_ref_629o
        ),
}


# ======================================================================================
# 7) G2 weighted reference audit
# ======================================================================================

g2_values_629o = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

g2_weights_629o = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

if (
    g2_values_629o.shape
    !=
    g2_weights_629o.shape
):
    raise RuntimeError(
        "Historical G2 values/weights shape mismatch."
    )

g2_audit_629o = {
    "shape":
        tuple(
            int(x)
            for x in g2_values_629o.shape
        ),

    "values_finite":
        bool(
            np.isfinite(
                g2_values_629o
            ).all()
        ),

    "weights_finite":
        bool(
            np.isfinite(
                g2_weights_629o
            ).all()
        ),

    "weights_nonnegative":
        bool(
            np.all(
                g2_weights_629o
                >= 0.0
            )
        ),

    "weight_sum":
        float(
            g2_weights_629o.sum()
        ),

    "value_minimum":
        float(
            g2_values_629o.min()
        ),

    "value_maximum":
        float(
            g2_values_629o.max()
        ),

    "unique_value_count":
        int(
            len(
                np.unique(
                    g2_values_629o
                )
            )
        ),

    "values_sha256":
        sha_array_629o(
            g2_values_629o
        ),

    "weights_sha256":
        sha_array_629o(
            g2_weights_629o
        ),
}


# ======================================================================================
# 8) Look only at explicitly referenced globals from the corrected phase helper
#
# IMPORTANT:
# We do NOT broadly scan globals().
# We inspect only global names literally referenced by the helper bytecode.
# ======================================================================================

phase_helper_629o = (
    apply_phase_adjustment_628fbr1
)

phase_global_names_629o = []

if hasattr(
    phase_helper_629o,
    "__code__",
):

    for name_629o in (
        phase_helper_629o
        .__code__
        .co_names
    ):

        if name_629o in globals():
            phase_global_names_629o.append(
                name_629o
            )


phase_global_inventory_629o = {}

for name_629o in sorted(
    set(
        phase_global_names_629o
    )
):

    # Exclude libraries/modules/classes from expansion.
    obj_629o = globals()[
        name_629o
    ]

    if inspect.ismodule(
        obj_629o
    ):
        continue

    if callable(
        obj_629o
    ):
        phase_global_inventory_629o[
            name_629o
        ] = {
            "kind":
                "callable",

            "signature":
                (
                    str(
                        inspect.signature(
                            obj_629o
                        )
                    )
                    if not inspect.isclass(
                        obj_629o
                    )
                    else "<class>"
                ),
        }

    else:
        phase_global_inventory_629o[
            name_629o
        ] = safe_metadata_629o(
            obj_629o
        )


# ======================================================================================
# 9) Audit contract
# ======================================================================================

audit_payload_629o = json.dumps(
    {
        "recipe_fbr1":
            recipe_fbr1_629o,

        "uncertainty_contract":
            uncertainty_contract_629o,

        "helper_source_hashes":
            {
                name: record[
                    "source_sha256"
                ]
                for name, record
                in helper_records_629o.items()
            },

        "g1_reference":
            g1_audit_629o,

        "g2_reference":
            g2_audit_629o,

        "phase_referenced_globals":
            phase_global_inventory_629o,

        "new_fd003_risk_created":
            False,

        "dev_scoring_performed":
            False,

        "dev_labels_used":
            False,

        "model_execution_performed":
            False,

        "mc_execution_performed":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    default=str,
    separators=(",", ":"),
).encode(
    "utf-8"
)

FD003_RISK_REFERENCE_RECOVERY_SHA256_629O = hashlib.sha256(
    audit_payload_629o
).hexdigest()


FD003_NEW_RISK_CREATED_629O = False
FD003_DEV_SCORING_PERFORMED_629O = False
FD003_DEV_LABELS_USED_629O = False
FD003_MODEL_EXECUTION_PERFORMED_629O = False
FD003_MC_EXECUTION_PERFORMED_629O = False
FD003_THRESHOLD_SELECTED_629O = False
FD003_TEST_CONTENTS_READ_629O = False
FD003_RUL_CONTENTS_READ_629O = False

all_checks_passed_629o = True


# ======================================================================================
# 10) Reporting
# ======================================================================================

print("=" * 130)
print(
    "STEP 6.29O — FD003 HISTORICAL UNCERTAINTY-RISK "
    "REFERENCE-CONSTRUCTION RECOVERY"
)
print("=" * 130)

print(
    "New FD003 risk created         :",
    False,
)

print(
    "DEV scoring performed          :",
    False,
)

print(
    "DEV labels used                :",
    False,
)

print(
    "Model execution performed      :",
    False,
)

print(
    "MC-dropout execution performed :",
    False,
)

print(
    "Threshold selected             :",
    False,
)

print(
    "TEST contents read             :",
    False,
)

print(
    "Official RUL contents read     :",
    False,
)


print(
    "\n--- CORRECTED FD002 UNCERTAINTY RECIPE (628FBR1) ---"
)

print(
    json.dumps(
        recipe_fbr1_629o,
        indent=2,
        sort_keys=True,
        default=str,
    )
)


print(
    "\n--- FD002 UNCERTAINTY-ONLY CONTRACT ---"
)

print(
    json.dumps(
        uncertainty_contract_629o,
        indent=2,
        sort_keys=True,
        default=str,
    )
)


print(
    "\n--- CORRECTED PHASE-ADJUSTMENT IMPLEMENTATION ---"
)

phase_record_629o = helper_records_629o[
    "apply_phase_adjustment_628fbr1"
]

print(
    "signature       :",
    phase_record_629o[
        "signature"
    ],
)

print(
    "source SHA-256  :",
    phase_record_629o[
        "source_sha256"
    ],
)

print(
    "referenced names:",
    phase_record_629o[
        "referenced_global_names"
    ],
)

if phase_record_629o[
    "source_available"
]:
    print(
        "\nSOURCE:\n"
    )
    print(
        phase_record_629o[
            "source"
        ]
    )


print(
    "\n--- PHASE-HELPER REFERENCED GLOBALS ---"
)

print(
    json.dumps(
        phase_global_inventory_629o,
        indent=2,
        sort_keys=True,
        default=str,
    )
)


print(
    "\n--- G1 REFERENCE AUDIT ---"
)

for key_629o, value_629o in (
    g1_audit_629o.items()
):
    print(
        f"{key_629o:26s}: "
        f"{value_629o}"
    )


print(
    "\n--- G2 WEIGHTED REFERENCE AUDIT ---"
)

for key_629o, value_629o in (
    g2_audit_629o.items()
):
    print(
        f"{key_629o:26s}: "
        f"{value_629o}"
    )


print(
    "\n--- WRAPPER CHAIN ---"
)

for name_629o in [
    "fd002_phase_adjusted_uncertainty_628hc0r2",
    "fd002_g1_uncertainty_risk_628hc0r2",
    "fd002_final_selective_risk_628hc0r2",
]:

    record_629o = helper_records_629o[
        name_629o
    ]

    print(
        f"\n{name_629o}"
    )

    print(
        "signature      :",
        record_629o[
            "signature"
        ],
    )

    print(
        "source SHA-256 :",
        record_629o[
            "source_sha256"
        ],
    )

    if record_629o[
        "source_available"
    ]:
        print(
            record_629o[
                "source"
            ]
        )


print(
    "\nReference-recovery contract SHA-256 :",
    FD003_RISK_REFERENCE_RECOVERY_SHA256_629O,
)

print(
    "\nPASS — corrected historical uncertainty-risk reference "
    "construction has been recovered structurally without creating "
    "an FD003 risk or selecting an FD003 threshold."
)

print(
    "Next: use this evidence to define the FD003-specific FIT-only "
    "uncertainty-calibration reference contract. Historical FD002 "
    "numeric references will NOT be transferred to FD003."
)

In [ ]:
# STEP 6.29P
# FD003 — strict historical phase-edge construction verification
#
# PURPOSE:
#   Resolve the one remaining low-level ambiguity before freezing the
#   FD003-specific FIT-only uncertainty calibration protocol:
#   exactly how the 5 deterministic-prediction phase edges were constructed.
#
# NO model inference / NO MC execution / NO DEV scoring / NO labels
# NO TEST / NO official RUL / NO FD003 risk / NO threshold selection

import hashlib
import numpy as np
import pandas as pd

# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629o", False)):
    raise RuntimeError(
        "STEP 6.29O must pass before STEP 6.29P."
    )

required_629p = [
    "FD002_PHASE_EDGES_628FBR1",
    "FD002_PHASE_REFERENCES_628FBR1",
]

missing_629p = [
    name
    for name in required_629p
    if name not in globals()
]

if missing_629p:
    raise RuntimeError(
        "Missing required recovered historical object(s): "
        + ", ".join(missing_629p)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629O", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29O."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629O", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29O."
    )


# ======================================================================================
# 2) Recovered edge/reference contract
# ======================================================================================

historical_edges_629p = np.asarray(
    FD002_PHASE_EDGES_628FBR1,
    dtype=np.float64,
).reshape(-1)

if historical_edges_629p.shape != (4,):
    raise RuntimeError(
        "Recovered FD002 phase-edge vector is not length 4."
    )

if not np.isfinite(historical_edges_629p).all():
    raise RuntimeError(
        "Recovered FD002 phase edges contain non-finite values."
    )

reference_lengths_629p = {
    int(bin_id_629p): int(
        len(
            np.asarray(
                ref_629p
            ).reshape(-1)
        )
    )
    for bin_id_629p, ref_629p
    in FD002_PHASE_REFERENCES_628FBR1.items()
}

if sorted(reference_lengths_629p.keys()) != [0, 1, 2, 3, 4]:
    raise RuntimeError(
        "Recovered FD002 phase references do not contain exactly bins 0..4."
    )

expected_fit_windows_629p = int(
    sum(
        reference_lengths_629p.values()
    )
)


# ======================================================================================
# 3) Strict filtered NAME-ONLY inventory for historical FIT prediction arrays
# ======================================================================================

forbidden_name_tokens_629p = (
    "test",
    "rul",
    "label",
    "target",
    "error",
    "mae",
    "score",
    "scoring",
)

candidate_names_629p = []

for name_629p in sorted(globals().keys()):
    lower_629p = name_629p.lower()

    historical_629p = (
        "628" in lower_629p
    )

    fit_629p = (
        "fit" in lower_629p
    )

    prediction_629p = (
        "pred" in lower_629p
        or "prediction" in lower_629p
    )

    forbidden_629p = any(
        token_629p in lower_629p
        for token_629p in forbidden_name_tokens_629p
    )

    if (
        historical_629p
        and fit_629p
        and prediction_629p
        and not forbidden_629p
    ):
        candidate_names_629p.append(
            name_629p
        )


# ======================================================================================
# 4) Candidate structural audit
# ======================================================================================

def raw_sha256_629p(values):
    arr_629p = np.ascontiguousarray(
        np.asarray(values)
    )

    if arr_629p.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype historical candidate refused."
        )

    return hashlib.sha256(
        arr_629p.tobytes(order="C")
    ).hexdigest()


candidate_rows_629p = []
eligible_candidates_629p = {}

for name_629p in candidate_names_629p:
    obj_629p = globals()[name_629p]

    row_629p = {
        "name": name_629p,
        "type": type(obj_629p).__name__,
        "numeric_array_like": False,
        "shape": None,
        "dtype": None,
        "finite": None,
        "length_matches_fit_reference_total": False,
        "sha256": None,
    }

    try:
        arr_629p = np.asarray(
            obj_629p
        )
    except Exception:
        candidate_rows_629p.append(
            row_629p
        )
        continue

    numeric_629p = bool(
        np.issubdtype(
            arr_629p.dtype,
            np.number,
        )
    )

    row_629p["numeric_array_like"] = numeric_629p
    row_629p["shape"] = tuple(
        int(x)
        for x in arr_629p.shape
    )
    row_629p["dtype"] = str(
        arr_629p.dtype
    )

    if not numeric_629p:
        candidate_rows_629p.append(
            row_629p
        )
        continue

    arr1d_629p = np.asarray(
        arr_629p,
        dtype=np.float64,
    ).reshape(-1)

    row_629p["finite"] = bool(
        np.isfinite(
            arr1d_629p
        ).all()
    )

    row_629p[
        "length_matches_fit_reference_total"
    ] = bool(
        len(arr1d_629p)
        ==
        expected_fit_windows_629p
    )

    row_629p["sha256"] = raw_sha256_629p(
        arr1d_629p
    )

    if (
        row_629p["finite"]
        and row_629p[
            "length_matches_fit_reference_total"
        ]
    ):
        eligible_candidates_629p[
            name_629p
        ] = arr1d_629p

    candidate_rows_629p.append(
        row_629p
    )


FD003_HISTORICAL_FIT_PREDICTION_CANDIDATES_629P = pd.DataFrame(
    candidate_rows_629p
)


# ======================================================================================
# 5) Test exact quintile-edge conventions against recovered historical edges
# ======================================================================================

quantile_probs_629p = np.array(
    [0.20, 0.40, 0.60, 0.80],
    dtype=np.float64,
)

quantile_methods_629p = [
    "linear",
    "lower",
    "higher",
    "midpoint",
    "nearest",
]

method_rows_629p = []

for name_629p, pred_629p in eligible_candidates_629p.items():

    historical_bin_ids_629p = np.digitize(
        pred_629p,
        historical_edges_629p,
        right=True,
    )

    historical_bin_counts_629p = [
        int(
            np.sum(
                historical_bin_ids_629p
                ==
                bin_id_629p
            )
        )
        for bin_id_629p in range(5)
    ]

    reference_counts_629p = [
        reference_lengths_629p[
            bin_id_629p
        ]
        for bin_id_629p in range(5)
    ]

    bin_counts_match_629p = bool(
        historical_bin_counts_629p
        ==
        reference_counts_629p
    )

    for method_629p in quantile_methods_629p:

        candidate_edges_629p = np.quantile(
            pred_629p,
            quantile_probs_629p,
            method=method_629p,
        ).astype(
            np.float64
        )

        exact_match_629p = bool(
            np.array_equal(
                candidate_edges_629p,
                historical_edges_629p,
            )
        )

        max_abs_diff_629p = float(
            np.max(
                np.abs(
                    candidate_edges_629p
                    -
                    historical_edges_629p
                )
            )
        )

        close_match_629p = bool(
            np.allclose(
                candidate_edges_629p,
                historical_edges_629p,
                rtol=0.0,
                atol=1e-12,
            )
        )

        method_rows_629p.append(
            {
                "candidate_name":
                    name_629p,

                "quantile_method":
                    method_629p,

                "exact_edge_match":
                    exact_match_629p,

                "close_edge_match_atol_1e-12":
                    close_match_629p,

                "max_abs_edge_difference":
                    max_abs_diff_629p,

                "historical_bin_counts":
                    historical_bin_counts_629p,

                "phase_reference_lengths":
                    reference_counts_629p,

                "bin_counts_match_reference_lengths":
                    bin_counts_match_629p,
            }
        )


FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P = pd.DataFrame(
    method_rows_629p
)


# ======================================================================================
# 6) Resolution status
# ======================================================================================

if len(FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P) > 0:

    exact_rows_629p = (
        FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P[
            FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P[
                "exact_edge_match"
            ]
        ]
    )

    close_rows_629p = (
        FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P[
            FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P[
                "close_edge_match_atol_1e-12"
            ]
        ]
    )

else:

    exact_rows_629p = pd.DataFrame()
    close_rows_629p = pd.DataFrame()


FD003_HISTORICAL_PHASE_EDGE_EXACT_MATCH_COUNT_629P = int(
    len(
        exact_rows_629p
    )
)

FD003_HISTORICAL_PHASE_EDGE_CLOSE_MATCH_COUNT_629P = int(
    len(
        close_rows_629p
    )
)

FD003_HISTORICAL_PHASE_EDGE_METHOD_RESOLVED_629P = bool(
    FD003_HISTORICAL_PHASE_EDGE_EXACT_MATCH_COUNT_629P
    >= 1
)


# ======================================================================================
# 7) Safety-state lock
# ======================================================================================

FD003_MODEL_EXECUTION_PERFORMED_629P = False
FD003_MC_EXECUTION_PERFORMED_629P = False
FD003_DEV_SCORING_PERFORMED_629P = False
FD003_DEV_LABELS_USED_629P = False
FD003_NEW_RISK_CREATED_629P = False
FD003_THRESHOLD_SELECTED_629P = False
FD003_TEST_CONTENTS_READ_629P = False
FD003_RUL_CONTENTS_READ_629P = False

all_checks_passed_629p = True


# ======================================================================================
# 8) Output
# ======================================================================================

print("=" * 130)
print(
    "STEP 6.29P — FD003 HISTORICAL PHASE-EDGE "
    "CONSTRUCTION VERIFICATION"
)
print("=" * 130)

print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("DEV labels used                :", False)
print("FD003 risk created             :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- RECOVERED HISTORICAL PHASE CONTRACT ---")

print(
    "Phase edge count               :",
    len(
        historical_edges_629p
    ),
)

print(
    "Phase reference lengths        :",
    [
        reference_lengths_629p[i]
        for i in range(5)
    ],
)

print(
    "Total FIT reference windows    :",
    expected_fit_windows_629p,
)

print("\n--- STRICT FIT-PREDICTION CANDIDATE INVENTORY ---")

if len(
    FD003_HISTORICAL_FIT_PREDICTION_CANDIDATES_629P
) == 0:

    print(
        "<no historical FIT prediction candidates found>"
    )

else:

    print(
        FD003_HISTORICAL_FIT_PREDICTION_CANDIDATES_629P.to_string(
            index=False
        )
    )

print("\n--- QUINTILE EDGE METHOD AUDIT ---")

if len(
    FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P
) == 0:

    print(
        "<no eligible historical FIT prediction vector available>"
    )

else:

    print(
        FD003_HISTORICAL_PHASE_EDGE_METHOD_AUDIT_629P.to_string(
            index=False
        )
    )

print("\n--- RESOLUTION ---")

print(
    "Exact matching candidate/method rows :",
    FD003_HISTORICAL_PHASE_EDGE_EXACT_MATCH_COUNT_629P,
)

print(
    "Close matching rows (1e-12)          :",
    FD003_HISTORICAL_PHASE_EDGE_CLOSE_MATCH_COUNT_629P,
)

print(
    "Exact historical edge rule resolved :",
    FD003_HISTORICAL_PHASE_EDGE_METHOD_RESOLVED_629P,
)

print(
    "\nPASS — this step only tested the recovered historical "
    "FIT phase-edge construction. No FD003 calibration/risk "
    "or threshold has been created."
)

print(
    "Next action depends on the exact-match result. "
    "Do not execute FIT MC calibration yet."
)

In [ ]:
# STEP 6.29P-R1
# Exact historical phase-edge floating-convention probe
#
# READ ONLY:
#   - historical FD002 FIT deterministic prediction
#   - recovered historical phase edges
#
# NO model inference / NO MC / NO DEV scoring / NO labels
# NO TEST / NO official RUL
# NO FD003 calibration / risk / threshold creation

import hashlib
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629p", False)):
    raise RuntimeError(
        "STEP 6.29P must pass before STEP 6.29P-R1."
    )

required_629pr1 = [
    "FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1",
    "FD002_PHASE_EDGES_628FBR1",
]

missing_629pr1 = [
    name
    for name in required_629pr1
    if name not in globals()
]

if missing_629pr1:
    raise RuntimeError(
        "Missing required historical object(s): "
        + ", ".join(missing_629pr1)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629P", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29P."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629P", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29P."
    )


# ======================================================================================
# 2) Historical source arrays
# ======================================================================================

pred64_629pr1 = np.asarray(
    FIT_DETERMINISTIC_PREDICTIONS_FD002_628FBR1,
    dtype=np.float64,
).reshape(-1)

historical_edges_629pr1 = np.asarray(
    FD002_PHASE_EDGES_628FBR1,
    dtype=np.float64,
).reshape(-1)

if pred64_629pr1.shape != (37852,):
    raise RuntimeError(
        "Historical FD002 FIT prediction length is not 37852."
    )

if historical_edges_629pr1.shape != (4,):
    raise RuntimeError(
        "Historical phase-edge vector is not length 4."
    )

if not np.isfinite(pred64_629pr1).all():
    raise RuntimeError(
        "Historical FIT prediction contains non-finite values."
    )

if not np.isfinite(historical_edges_629pr1).all():
    raise RuntimeError(
        "Historical phase edges contain non-finite values."
    )

q_629pr1 = np.asarray(
    [0.20, 0.40, 0.60, 0.80],
    dtype=np.float64,
)

pct_629pr1 = np.asarray(
    [20.0, 40.0, 60.0, 80.0],
    dtype=np.float64,
)


# ======================================================================================
# 3) Candidate floating pathways
# ======================================================================================

candidate_edges_629pr1 = {}

# A — canonical float64 quantile
candidate_edges_629pr1[
    "float64_np_quantile_linear"
] = np.asarray(
    np.quantile(
        pred64_629pr1,
        q_629pr1,
        method="linear",
    ),
    dtype=np.float64,
)

# B — float64 percentile
candidate_edges_629pr1[
    "float64_np_percentile_linear"
] = np.asarray(
    np.percentile(
        pred64_629pr1,
        pct_629pr1,
        method="linear",
    ),
    dtype=np.float64,
)

# C — float32 source, quantile result promoted to float64
pred32_629pr1 = np.asarray(
    pred64_629pr1,
    dtype=np.float32,
)

candidate_edges_629pr1[
    "float32_np_quantile_linear"
] = np.asarray(
    np.quantile(
        pred32_629pr1,
        q_629pr1,
        method="linear",
    ),
    dtype=np.float64,
)

# D — float32 source, percentile result promoted to float64
candidate_edges_629pr1[
    "float32_np_percentile_linear"
] = np.asarray(
    np.percentile(
        pred32_629pr1,
        pct_629pr1,
        method="linear",
    ),
    dtype=np.float64,
)

# E/F — explicit manual linear interpolation, two algebraically
# equivalent operation orders.
sorted64_629pr1 = np.sort(
    pred64_629pr1
)

n_629pr1 = int(
    len(sorted64_629pr1)
)

manual_a_629pr1 = []
manual_b_629pr1 = []

for q_value_629pr1 in q_629pr1:

    h_629pr1 = (
        (n_629pr1 - 1)
        *
        float(q_value_629pr1)
    )

    lo_629pr1 = int(
        np.floor(h_629pr1)
    )

    hi_629pr1 = int(
        np.ceil(h_629pr1)
    )

    frac_629pr1 = np.float64(
        h_629pr1 - lo_629pr1
    )

    a_629pr1 = np.float64(
        sorted64_629pr1[lo_629pr1]
    )

    b_629pr1 = np.float64(
        sorted64_629pr1[hi_629pr1]
    )

    value_a_629pr1 = np.float64(
        a_629pr1
        +
        frac_629pr1
        *
        (
            b_629pr1
            -
            a_629pr1
        )
    )

    value_b_629pr1 = np.float64(
        (
            np.float64(1.0)
            -
            frac_629pr1
        )
        *
        a_629pr1
        +
        frac_629pr1
        *
        b_629pr1
    )

    manual_a_629pr1.append(
        value_a_629pr1
    )

    manual_b_629pr1.append(
        value_b_629pr1
    )

candidate_edges_629pr1[
    "manual_a_plus_frac_delta"
] = np.asarray(
    manual_a_629pr1,
    dtype=np.float64,
)

candidate_edges_629pr1[
    "manual_weighted_sum"
] = np.asarray(
    manual_b_629pr1,
    dtype=np.float64,
)


# ======================================================================================
# 4) ULP-distance helper for finite nonnegative float64 edges
# ======================================================================================

def ulp_distance_629pr1(a, b):

    a_629pr1 = np.asarray(
        a,
        dtype=np.float64,
    )

    b_629pr1 = np.asarray(
        b,
        dtype=np.float64,
    )

    if not (
        np.isfinite(a_629pr1).all()
        and
        np.isfinite(b_629pr1).all()
    ):
        raise RuntimeError(
            "ULP audit requires finite values."
        )

    if bool(
        (a_629pr1 < 0.0).any()
        or
        (b_629pr1 < 0.0).any()
    ):
        raise RuntimeError(
            "This simple ULP audit expects nonnegative phase edges."
        )

    a_bits_629pr1 = (
        a_629pr1
        .view(np.uint64)
    )

    b_bits_629pr1 = (
        b_629pr1
        .view(np.uint64)
    )

    distance_629pr1 = np.where(
        a_bits_629pr1 >= b_bits_629pr1,
        a_bits_629pr1 - b_bits_629pr1,
        b_bits_629pr1 - a_bits_629pr1,
    )

    return distance_629pr1


# ======================================================================================
# 5) Compare every candidate with the persisted historical edges
# ======================================================================================

rows_629pr1 = []

for (
    method_name_629pr1,
    edges_629pr1,
) in candidate_edges_629pr1.items():

    edges_629pr1 = np.asarray(
        edges_629pr1,
        dtype=np.float64,
    ).reshape(-1)

    exact_629pr1 = bool(
        np.array_equal(
            edges_629pr1,
            historical_edges_629pr1,
        )
    )

    abs_diff_629pr1 = np.abs(
        edges_629pr1
        -
        historical_edges_629pr1
    )

    ulps_629pr1 = ulp_distance_629pr1(
        edges_629pr1,
        historical_edges_629pr1,
    )

    phase_ids_629pr1 = np.digitize(
        pred64_629pr1,
        edges_629pr1,
        right=True,
    )

    phase_counts_629pr1 = [
        int(
            np.sum(
                phase_ids_629pr1
                ==
                phase_id_629pr1
            )
        )
        for phase_id_629pr1 in range(5)
    ]

    rows_629pr1.append(
        {
            "method":
                method_name_629pr1,

            "exact_match":
                exact_629pr1,

            "max_abs_difference":
                float(
                    np.max(
                        abs_diff_629pr1
                    )
                ),

            "max_ULP_distance":
                int(
                    np.max(
                        ulps_629pr1
                    )
                ),

            "per_edge_ULP_distance":
                [
                    int(x)
                    for x in ulps_629pr1
                ],

            "phase_counts":
                phase_counts_629pr1,

            "phase_counts_match":
                bool(
                    phase_counts_629pr1
                    ==
                    [7571, 7570, 7570, 7570, 7571]
                ),
        }
    )


FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1 = pd.DataFrame(
    rows_629pr1
)


# ======================================================================================
# 6) Historical-edge hexadecimal representation
# ======================================================================================

edge_detail_rows_629pr1 = []

linear_edges_629pr1 = candidate_edges_629pr1[
    "float64_np_quantile_linear"
]

linear_ulps_629pr1 = ulp_distance_629pr1(
    linear_edges_629pr1,
    historical_edges_629pr1,
)

for edge_index_629pr1 in range(4):

    historical_value_629pr1 = np.float64(
        historical_edges_629pr1[
            edge_index_629pr1
        ]
    )

    linear_value_629pr1 = np.float64(
        linear_edges_629pr1[
            edge_index_629pr1
        ]
    )

    edge_detail_rows_629pr1.append(
        {
            "edge":
                int(edge_index_629pr1 + 1),

            "historical_value":
                float(historical_value_629pr1),

            "linear_quantile_value":
                float(linear_value_629pr1),

            "absolute_difference":
                float(
                    abs(
                        historical_value_629pr1
                        -
                        linear_value_629pr1
                    )
                ),

            "ULP_distance":
                int(
                    linear_ulps_629pr1[
                        edge_index_629pr1
                    ]
                ),

            "historical_hex":
                historical_value_629pr1.hex(),

            "linear_quantile_hex":
                linear_value_629pr1.hex(),
        }
    )


FD003_PHASE_EDGE_HEX_AUDIT_629PR1 = pd.DataFrame(
    edge_detail_rows_629pr1
)


# ======================================================================================
# 7) Conservative resolution logic
# ======================================================================================

exact_methods_629pr1 = (
    FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1[
        FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1[
            "exact_match"
        ]
    ]
)

minimum_max_ulp_629pr1 = int(
    FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1[
        "max_ULP_distance"
    ].min()
)

linear_row_629pr1 = (
    FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1[
        FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1[
            "method"
        ]
        ==
        "float64_np_quantile_linear"
    ]
    .iloc[0]
)

linear_operationally_identical_629pr1 = bool(
    linear_row_629pr1[
        "phase_counts_match"
    ]
)

linear_within_tiny_float_difference_629pr1 = bool(
    float(
        linear_row_629pr1[
            "max_abs_difference"
        ]
    )
    <=
    1e-12
)

FD003_PHASE_EDGE_EXACT_FLOATING_PATH_RESOLVED_629PR1 = bool(
    len(
        exact_methods_629pr1
    )
    >= 1
)

FD003_PHASE_EDGE_LINEAR_QUINTILE_SEMANTICS_CONFIRMED_629PR1 = bool(
    linear_operationally_identical_629pr1
    and
    linear_within_tiny_float_difference_629pr1
)


# ======================================================================================
# 8) Audit fingerprint
# ======================================================================================

audit_payload_629pr1 = json.dumps(
    {
        "historical_edges":
            [
                float(x)
                for x in historical_edges_629pr1
            ],

        "historical_fit_prediction_sha256":
            hashlib.sha256(
                np.ascontiguousarray(
                    pred64_629pr1
                ).tobytes(order="C")
            ).hexdigest(),

        "candidate_results":
            rows_629pr1,

        "exact_floating_path_resolved":
            FD003_PHASE_EDGE_EXACT_FLOATING_PATH_RESOLVED_629PR1,

        "linear_quintile_semantics_confirmed":
            FD003_PHASE_EDGE_LINEAR_QUINTILE_SEMANTICS_CONFIRMED_629PR1,

        "new_fd003_calibration_created":
            False,

        "model_execution_performed":
            False,

        "mc_execution_performed":
            False,

        "dev_scoring_performed":
            False,

        "dev_labels_used":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_PHASE_EDGE_FLOATING_AUDIT_SHA256_629PR1 = hashlib.sha256(
    audit_payload_629pr1
).hexdigest()


FD003_MODEL_EXECUTION_PERFORMED_629PR1 = False
FD003_MC_EXECUTION_PERFORMED_629PR1 = False
FD003_DEV_SCORING_PERFORMED_629PR1 = False
FD003_DEV_LABELS_USED_629PR1 = False
FD003_NEW_RISK_CREATED_629PR1 = False
FD003_THRESHOLD_SELECTED_629PR1 = False
FD003_TEST_CONTENTS_READ_629PR1 = False
FD003_RUL_CONTENTS_READ_629PR1 = False

all_checks_passed_629pr1 = True


# ======================================================================================
# 9) Output
# ======================================================================================

print("=" * 132)

print(
    "STEP 6.29P-R1 — FD003 HISTORICAL PHASE-EDGE "
    "FLOATING-CONVENTION PROBE"
)

print("=" * 132)

print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("DEV labels used                :", False)
print("FD003 calibration created      :", False)
print("FD003 risk created             :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FLOATING PATHWAY AUDIT ---"
)

print(
    FD003_PHASE_EDGE_FLOATING_AUDIT_629PR1.to_string(
        index=False
    )
)

print(
    "\n--- LINEAR-QUANTILE EDGE DETAIL ---"
)

print(
    FD003_PHASE_EDGE_HEX_AUDIT_629PR1.to_string(
        index=False
    )
)

print(
    "\n--- CONSERVATIVE RESOLUTION ---"
)

print(
    "Exact floating pathway resolved     :",
    FD003_PHASE_EDGE_EXACT_FLOATING_PATH_RESOLVED_629PR1,
)

print(
    "Minimum candidate max ULP distance  :",
    minimum_max_ulp_629pr1,
)

print(
    "Linear quintile phase counts exact  :",
    linear_operationally_identical_629pr1,
)

print(
    "Linear max abs diff <= 1e-12        :",
    linear_within_tiny_float_difference_629pr1,
)

print(
    "Linear-quintile semantics confirmed :",
    FD003_PHASE_EDGE_LINEAR_QUINTILE_SEMANTICS_CONFIRMED_629PR1,
)

print(
    "Audit SHA-256                       :",
    FD003_PHASE_EDGE_FLOATING_AUDIT_SHA256_629PR1,
)

print(
    "\nPASS — only the historical floating-point construction "
    "of the four phase edges was investigated."
)

print(
    "No FD003 calibration, risk or threshold has been created. "
    "Do not execute FIT MC calibration until this output is reviewed."
)

In [ ]:
# STEP 6.29Q
# FD003 FIT-only uncertainty-calibration protocol freeze
# NO MODEL INFERENCE / NO MC EXECUTION / NO DEV SCORING / NO TEST OR RUL ACCESS

import hashlib
import json

# --------------------------------------------------------------------------------------
# 1) Preconditions
# --------------------------------------------------------------------------------------

if not bool(globals().get("all_checks_passed_629pr1", False)):
    raise RuntimeError(
        "STEP 6.29P-R1 must pass before freezing the FD003 calibration protocol."
    )

required_629q = [
    "FD003_TRAINED_MODEL_STATE_SHA256_629K",
    "FD003_TRAINING_RUN_CONTRACT_SHA256_629K",
    "FD003_MC_DEV_CONTRACT_SHA256_629M",
    "FD003_PHASE_EDGE_FLOATING_AUDIT_SHA256_629PR1",
    "FD003_PHASE_EDGE_EXACT_FLOATING_PATH_RESOLVED_629PR1",
    "FD003_PHASE_EDGE_LINEAR_QUINTILE_SEMANTICS_CONFIRMED_629PR1",
]

missing_629q = [
    name
    for name in required_629q
    if name not in globals()
]

if missing_629q:
    raise RuntimeError(
        "Missing required provenance object(s): "
        + ", ".join(missing_629q)
    )

if bool(FD003_PHASE_EDGE_EXACT_FLOATING_PATH_RESOLVED_629PR1):
    raise RuntimeError(
        "Unexpected state: P-R1 reported exact floating-path recovery. "
        "Review before using the documented limitation path."
    )

if not bool(FD003_PHASE_EDGE_LINEAR_QUINTILE_SEMANTICS_CONFIRMED_629PR1):
    raise RuntimeError(
        "Linear-quintile phase semantics were not confirmed."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629PR1", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29P-R1."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629PR1", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29P-R1."
    )

# --------------------------------------------------------------------------------------
# 2) Prospectively freeze the FD003-specific phase-calibration rule
# --------------------------------------------------------------------------------------

FD003_PHASE_CALIBRATION_SOURCE_629Q = "FD003_FIT_only"
FD003_PHASE_SOURCE_629Q = "deterministic_prediction"

FD003_PHASE_REQUESTED_BINS_629Q = 5
FD003_PHASE_QUANTILE_PROBS_629Q = [0.20, 0.40, 0.60, 0.80]
FD003_PHASE_QUANTILE_METHOD_629Q = "linear"
FD003_PHASE_EDGE_DTYPE_629Q = "float64"
FD003_PHASE_ASSIGNMENT_629Q = "np.digitize(right=True)"

FD003_BASE_UNCERTAINTY_629Q = "MC_dropout_STD"
FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q = 50
FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q = 42
FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q = 512
FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q = 0

FD003_WITHIN_PHASE_MAPPING_629Q = "empirical_midrank_percentile"

FD003_HISTORICAL_FLOATING_LIMITATION_629Q = {
    "exact_historical_floating_path_recovered": False,
    "linear_quintile_semantics_confirmed": True,
    "minimum_candidate_max_ulp_distance": 1,
    "historical_linear_max_abs_difference": 1.4210854715202004e-14,
    "interpretation":
        "Historical phase-edge semantics are recovered, but one persisted FD002 "
        "edge differs by 1 ULP from all tested equivalent linear-quantile pathways. "
        "FD003 therefore freezes an explicit prospective NumPy float64 "
        "quantile(method='linear') convention and does not claim bit-exact "
        "historical floating-point reproduction.",
}

# --------------------------------------------------------------------------------------
# 3) Freeze calibration/reference roles
# --------------------------------------------------------------------------------------

FD003_UNCERTAINTY_REFERENCE_ROLE_CONTRACT_629Q = {
    "phase_calibration_reference":
        "FD003_FIT_only",

    "phase_calibration_uses_labels":
        False,

    "phase_calibration_point_prediction":
        "deterministic trained-model prediction on FD003 FIT windows",

    "phase_calibration_uncertainty":
        "50-pass MC-dropout STD on the same FD003 FIT windows",

    "phase_edges":
        "four deterministic-prediction quintile edges at q=[0.2,0.4,0.6,0.8]",

    "phase_assignment":
        "np.digitize(prediction, edges, right=True)",

    "within_phase_mapping":
        "empirical_midrank_percentile of FIT MC-dropout STD",

    "g1_reference_status":
        "not_created_yet",

    "g2_reference_status":
        "not_created_yet",

    "threshold_status":
        "not_selected",
}

# --------------------------------------------------------------------------------------
# 4) Protocol contract + fingerprint
# --------------------------------------------------------------------------------------

FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_629Q = {
    "step":
        "6.29Q",

    "dataset":
        "FD003",

    "trained_model_state_sha256":
        FD003_TRAINED_MODEL_STATE_SHA256_629K,

    "training_run_contract_sha256":
        FD003_TRAINING_RUN_CONTRACT_SHA256_629K,

    "clean_dev_mc_contract_sha256":
        FD003_MC_DEV_CONTRACT_SHA256_629M,

    "historical_phase_edge_audit_sha256":
        FD003_PHASE_EDGE_FLOATING_AUDIT_SHA256_629PR1,

    "historical_floating_limitation":
        dict(FD003_HISTORICAL_FLOATING_LIMITATION_629Q),

    "phase_reference_source":
        FD003_PHASE_CALIBRATION_SOURCE_629Q,

    "phase_source":
        FD003_PHASE_SOURCE_629Q,

    "requested_bins":
        FD003_PHASE_REQUESTED_BINS_629Q,

    "quantile_probabilities":
        list(FD003_PHASE_QUANTILE_PROBS_629Q),

    "quantile_method":
        FD003_PHASE_QUANTILE_METHOD_629Q,

    "phase_edge_dtype":
        FD003_PHASE_EDGE_DTYPE_629Q,

    "phase_assignment":
        FD003_PHASE_ASSIGNMENT_629Q,

    "base_uncertainty":
        FD003_BASE_UNCERTAINTY_629Q,

    "mc_passes":
        FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q,

    "mc_seed":
        FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q,

    "mc_batch_size":
        FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q,

    "mc_std_ddof":
        FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q,

    "within_phase_mapping":
        FD003_WITHIN_PHASE_MAPPING_629Q,

    "reference_roles":
        dict(FD003_UNCERTAINTY_REFERENCE_ROLE_CONTRACT_629Q),

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "dev_labels_used":
        False,

    "fd003_risk_created":
        False,

    "threshold_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

protocol_payload_629q = json.dumps(
    FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_629Q,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_SHA256_629Q = hashlib.sha256(
    protocol_payload_629q
).hexdigest()

FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_LOCKED_629Q = True

FD003_MODEL_INFERENCE_PERFORMED_629Q = False
FD003_MC_EXECUTION_PERFORMED_629Q = False
FD003_DEV_SCORING_PERFORMED_629Q = False
FD003_DEV_LABELS_USED_629Q = False
FD003_NEW_RISK_CREATED_629Q = False
FD003_THRESHOLD_SELECTED_629Q = False
FD003_TEST_CONTENTS_READ_629Q = False
FD003_RUL_CONTENTS_READ_629Q = False

all_checks_passed_629q = True

# --------------------------------------------------------------------------------------
# 5) Output
# --------------------------------------------------------------------------------------

print("=" * 132)
print(
    "STEP 6.29Q — FD003 FIT-ONLY UNCERTAINTY-CALIBRATION "
    "PROTOCOL FREEZE"
)
print("=" * 132)

print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("DEV labels used                :", False)
print("FD003 risk created             :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- HISTORICAL PROVENANCE RESOLUTION ---")

print(
    "Exact historical floating path recovered :",
    False,
)

print(
    "Linear-quintile semantics confirmed       :",
    True,
)

print(
    "Historical minimum max ULP distance       :",
    1,
)

print(
    "Historical max abs edge difference        :",
    FD003_HISTORICAL_FLOATING_LIMITATION_629Q[
        "historical_linear_max_abs_difference"
    ],
)

print(
    "FD003 convention                          :",
    "np.quantile(float64, q=[.2,.4,.6,.8], method='linear')",
)

print("\n--- FROZEN FD003 FIT CALIBRATION RULE ---")

print(
    "Reference population          :",
    FD003_PHASE_CALIBRATION_SOURCE_629Q,
)

print(
    "Phase source                  :",
    FD003_PHASE_SOURCE_629Q,
)

print(
    "Requested bins                :",
    FD003_PHASE_REQUESTED_BINS_629Q,
)

print(
    "Phase assignment              :",
    FD003_PHASE_ASSIGNMENT_629Q,
)

print(
    "Base uncertainty              :",
    FD003_BASE_UNCERTAINTY_629Q,
)

print(
    "MC passes / seed / batch      :",
    (
        FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q,
        FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q,
        FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q,
    ),
)

print(
    "MC STD ddof                   :",
    FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q,
)

print(
    "Within-phase mapping          :",
    FD003_WITHIN_PHASE_MAPPING_629Q,
)

print(
    "G1 reference                  :",
    "NOT CREATED YET",
)

print(
    "G2 reference                  :",
    "NOT CREATED YET",
)

print(
    "Threshold                     :",
    "NOT SELECTED",
)

print(
    "\nCalibration-protocol SHA-256  :",
    FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_SHA256_629Q,
)

print(
    "\nPASS — the FD003-specific FIT-only uncertainty-calibration "
    "methodology is now prospectively frozen."
)

print(
    "The 1-ULP historical FD002 edge discrepancy is preserved as an "
    "explicit provenance limitation, not silently treated as exact recovery."
)

print(
    "Next: execute deterministic FIT inference and one canonical 50-pass "
    "FIT MC-dropout run under this frozen protocol, then build the five "
    "FD003 FIT phase uncertainty references. No DEV labels, TEST, RUL, "
    "risk threshold, or retuning will be used."
)

In [ ]:
# STEP 6.29R
# FD003 FIT-only deterministic prediction + canonical MC-dropout calibration execution
# + five phase uncertainty-reference freeze
#
# USES: FD003 FIT FEATURES ONLY
# DOES NOT USE: FIT labels, DEV labels, TEST, official RUL, scoring, risk threshold, retuning

import hashlib
import inspect
import json
import numpy as np
import torch


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629q", False)):
    raise RuntimeError(
        "STEP 6.29Q must pass before STEP 6.29R."
    )

required_629r = [
    "MODEL_FD003_629J",
    "FD003_DEVICE_629J",
    "FIT_X_WRITABLE_FD003_629JR1",
    "FD003_TRAINED_MODEL_STATE_SHA256_629K",
    "FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_SHA256_629Q",
    "FD003_PHASE_QUANTILE_PROBS_629Q",
    "FD003_PHASE_QUANTILE_METHOD_629Q",
    "FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q",
    "FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q",
    "FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q",
    "FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q",
    "mc_dropout_predict_fd003_629m",
]

missing_629r = [
    name
    for name in required_629r
    if name not in globals()
]

if missing_629r:
    raise RuntimeError(
        "Missing required FD003 FIT-calibration object(s): "
        + ", ".join(missing_629r)
    )

if bool(globals().get("FD003_FIT_CALIBRATION_EXECUTED_629R", False)):
    raise RuntimeError(
        "STEP 6.29R has already executed in this session. "
        "Do not rerun the canonical FIT calibration execution."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629Q", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29Q."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629Q", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29Q."
    )

if int(FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q) != 0:
    raise RuntimeError(
        "Frozen FD003 calibration contract requires MC STD ddof=0."
    )

if str(FD003_PHASE_QUANTILE_METHOD_629Q) != "linear":
    raise RuntimeError(
        "Frozen FD003 phase-edge contract requires quantile method='linear'."
    )


# ======================================================================================
# 2) Helpers
# ======================================================================================

def _raw_sha256_629r(value):

    arr_629r = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629r.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29R hashing."
        )

    return hashlib.sha256(
        arr_629r.tobytes(order="C")
    ).hexdigest()


def _state_dict_sha256_629r(model):

    hasher_629r = hashlib.sha256()

    state_629r = model.state_dict()

    for key_629r in sorted(
        state_629r.keys()
    ):

        tensor_629r = (
            state_629r[key_629r]
            .detach()
            .cpu()
            .contiguous()
        )

        key_bytes_629r = key_629r.encode("utf-8")

        dtype_bytes_629r = str(
            tensor_629r.dtype
        ).encode("utf-8")

        shape_bytes_629r = json.dumps(
            list(tensor_629r.shape),
            separators=(",", ":"),
        ).encode("utf-8")

        hasher_629r.update(
            len(key_bytes_629r).to_bytes(
                8,
                "little",
            )
        )
        hasher_629r.update(
            key_bytes_629r
        )

        hasher_629r.update(
            len(dtype_bytes_629r).to_bytes(
                8,
                "little",
            )
        )
        hasher_629r.update(
            dtype_bytes_629r
        )

        hasher_629r.update(
            len(shape_bytes_629r).to_bytes(
                8,
                "little",
            )
        )
        hasher_629r.update(
            shape_bytes_629r
        )

        hasher_629r.update(
            tensor_629r.numpy().tobytes(
                order="C"
            )
        )

    return hasher_629r.hexdigest()


@torch.inference_mode()
def _deterministic_predict_fit_629r(
    X,
    model,
    device,
    batch_size=512,
):

    X_629r = np.asarray(
        X,
        dtype=np.float32,
    )

    n_629r = int(
        len(X_629r)
    )

    prediction_629r = np.empty(
        n_629r,
        dtype=np.float64,
    )

    original_training_state_629r = bool(
        model.training
    )

    try:

        model.eval()

        for start_629r in range(
            0,
            n_629r,
            int(batch_size),
        ):

            end_629r = min(
                start_629r
                +
                int(batch_size),
                n_629r,
            )

            xb_629r = torch.as_tensor(
                X_629r[
                    start_629r:end_629r
                ],
                dtype=torch.float32,
                device=device,
            )

            pred_629r = (
                model(
                    xb_629r
                )
                .reshape(-1)
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            if len(
                pred_629r
            ) != (
                end_629r
                -
                start_629r
            ):
                raise RuntimeError(
                    "FD003 FIT deterministic prediction count mismatch."
                )

            prediction_629r[
                start_629r:end_629r
            ] = pred_629r

    finally:

        if original_training_state_629r:
            model.train()
        else:
            model.eval()

    return prediction_629r


# ======================================================================================
# 3) Verify canonical MC helper before reuse
# ======================================================================================

mc_helper_629r = (
    mc_dropout_predict_fd003_629m
)

if not callable(
    mc_helper_629r
):
    raise RuntimeError(
        "Canonical STEP 6.29M MC helper is not callable."
    )

mc_helper_signature_629r = str(
    inspect.signature(
        mc_helper_629r
    )
)

expected_mc_signature_629r = (
    "(X, model, device, passes=50, batch_size=512, seed=42)"
)

if (
    mc_helper_signature_629r
    !=
    expected_mc_signature_629r
):
    raise RuntimeError(
        "Canonical MC helper signature changed: "
        +
        mc_helper_signature_629r
    )

mc_helper_bytecode_sha256_629r = hashlib.sha256(
    mc_helper_629r.__code__.co_code
).hexdigest()


# ======================================================================================
# 4) FIT feature + trained-model integrity
# ======================================================================================

X_fit_629r = np.asarray(
    FIT_X_WRITABLE_FD003_629JR1,
    dtype=np.float32,
)

if X_fit_629r.shape != (
    17114,
    30,
    18,
):
    raise RuntimeError(
        "FD003 FIT window tensor shape changed before calibration."
    )

if not np.isfinite(
    X_fit_629r
).all():
    raise RuntimeError(
        "FD003 FIT features contain non-finite values."
    )

model_629r = (
    MODEL_FD003_629J
)

device_629r = (
    FD003_DEVICE_629J
)

model_sha_before_629r = _state_dict_sha256_629r(
    model_629r
)

if (
    model_sha_before_629r
    !=
    FD003_TRAINED_MODEL_STATE_SHA256_629K
):
    raise RuntimeError(
        "FD003 trained-model state changed before FIT calibration."
    )

if bool(
    model_629r.training
):
    raise RuntimeError(
        "FD003 trained model is expected to be in eval mode before FIT calibration."
    )


# ======================================================================================
# 5) Deterministic FIT inference
# ======================================================================================

FD003_FIT_CALIBRATION_EXECUTED_629R = True

FD003_FIT_DETERMINISTIC_PREDICTION_629R = (
    _deterministic_predict_fit_629r(
        X_fit_629r,
        model_629r,
        device_629r,
        batch_size=int(
            FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q
        ),
    )
)

if (
    FD003_FIT_DETERMINISTIC_PREDICTION_629R.shape
    !=
    (17114,)
):
    raise RuntimeError(
        "FD003 FIT deterministic prediction shape mismatch."
    )

if not np.isfinite(
    FD003_FIT_DETERMINISTIC_PREDICTION_629R
).all():
    raise RuntimeError(
        "FD003 FIT deterministic prediction contains non-finite values."
    )


# ======================================================================================
# 6) One canonical 50-pass FIT MC-dropout run
# ======================================================================================

(
    FD003_FIT_MC_MATRIX_629R,
    FD003_FIT_MC_MEAN_629R,
    FD003_FIT_MC_STD_629R,
    FD003_FIT_MC_RUNTIME_SECONDS_629R,
) = mc_helper_629r(
    X_fit_629r,
    model_629r,
    device_629r,
    passes=int(
        FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q
    ),
    batch_size=int(
        FD003_MC_BATCH_SIZE_FOR_FIT_CALIBRATION_629Q
    ),
    seed=int(
        FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q
    ),
)

if (
    FD003_FIT_MC_MATRIX_629R.shape
    !=
    (
        50,
        17114,
    )
):
    raise RuntimeError(
        "FD003 FIT MC matrix shape mismatch."
    )

if (
    FD003_FIT_MC_MATRIX_629R.dtype
    !=
    np.float32
):
    raise RuntimeError(
        "FD003 FIT MC matrix dtype is not float32."
    )

if (
    FD003_FIT_MC_MEAN_629R.shape
    !=
    (17114,)
):
    raise RuntimeError(
        "FD003 FIT MC mean shape mismatch."
    )

if (
    FD003_FIT_MC_STD_629R.shape
    !=
    (17114,)
):
    raise RuntimeError(
        "FD003 FIT MC STD shape mismatch."
    )

for name_629r, arr_629r in [
    (
        "MC matrix",
        FD003_FIT_MC_MATRIX_629R,
    ),
    (
        "MC mean",
        FD003_FIT_MC_MEAN_629R,
    ),
    (
        "MC STD",
        FD003_FIT_MC_STD_629R,
    ),
]:

    if not np.isfinite(
        arr_629r
    ).all():
        raise RuntimeError(
            f"FD003 FIT {name_629r} contains non-finite values."
        )

if bool(
    (
        FD003_FIT_MC_STD_629R
        <
        0.0
    ).any()
):
    raise RuntimeError(
        "FD003 FIT MC STD contains negative values."
    )

mc_mean_rederived_629r = (
    FD003_FIT_MC_MATRIX_629R
    .mean(
        axis=0
    )
    .astype(
        np.float64
    )
)

mc_std_rederived_629r = (
    FD003_FIT_MC_MATRIX_629R
    .std(
        axis=0,
        ddof=0,
    )
    .astype(
        np.float64
    )
)

if not np.array_equal(
    FD003_FIT_MC_MEAN_629R,
    mc_mean_rederived_629r,
):
    raise RuntimeError(
        "FD003 FIT MC mean is not exactly rederived from the matrix."
    )

if not np.array_equal(
    FD003_FIT_MC_STD_629R,
    mc_std_rederived_629r,
):
    raise RuntimeError(
        "FD003 FIT MC STD is not exactly rederived with ddof=0."
    )


# ======================================================================================
# 7) Construct five deterministic-prediction phases
# ======================================================================================

FD003_PHASE_EDGES_629R = np.asarray(
    np.quantile(
        np.asarray(
            FD003_FIT_DETERMINISTIC_PREDICTION_629R,
            dtype=np.float64,
        ),
        np.asarray(
            FD003_PHASE_QUANTILE_PROBS_629Q,
            dtype=np.float64,
        ),
        method="linear",
    ),
    dtype=np.float64,
)

if (
    FD003_PHASE_EDGES_629R.shape
    !=
    (4,)
):
    raise RuntimeError(
        "FD003 phase-edge vector is not length 4."
    )

if not np.isfinite(
    FD003_PHASE_EDGES_629R
).all():
    raise RuntimeError(
        "FD003 phase edges contain non-finite values."
    )

if not bool(
    np.all(
        np.diff(
            FD003_PHASE_EDGES_629R
        )
        >
        0.0
    )
):
    raise RuntimeError(
        "FD003 phase edges are not strictly increasing."
    )

FD003_FIT_PHASE_ID_629R = np.digitize(
    FD003_FIT_DETERMINISTIC_PREDICTION_629R,
    FD003_PHASE_EDGES_629R,
    right=True,
).astype(
    np.int64
)

phase_counts_629r = [
    int(
        np.sum(
            FD003_FIT_PHASE_ID_629R
            ==
            phase_629r
        )
    )
    for phase_629r in range(5)
]

if sum(
    phase_counts_629r
) != 17114:
    raise RuntimeError(
        "FD003 phase counts do not cover all FIT windows."
    )

if any(
    count_629r <= 0
    for count_629r in phase_counts_629r
):
    raise RuntimeError(
        "At least one FD003 FIT phase is empty."
    )


# ======================================================================================
# 8) Build five FIT-only sorted MC-STD references
# ======================================================================================

FD003_PHASE_REFERENCES_629R = {}

for phase_629r in range(5):

    mask_629r = (
        FD003_FIT_PHASE_ID_629R
        ==
        phase_629r
    )

    reference_629r = np.sort(
        np.asarray(
            FD003_FIT_MC_STD_629R[
                mask_629r
            ],
            dtype=np.float64,
        )
    )

    if len(
        reference_629r
    ) != phase_counts_629r[
        phase_629r
    ]:
        raise RuntimeError(
            f"FD003 phase {phase_629r} reference length mismatch."
        )

    if not np.isfinite(
        reference_629r
    ).all():
        raise RuntimeError(
            f"FD003 phase {phase_629r} reference contains non-finite values."
        )

    reference_629r.setflags(
        write=False
    )

    FD003_PHASE_REFERENCES_629R[
        int(
            phase_629r
        )
    ] = reference_629r


# ======================================================================================
# 9) Post-execution model integrity
# ======================================================================================

model_sha_after_629r = _state_dict_sha256_629r(
    model_629r
)

model_state_unchanged_629r = bool(
    model_sha_after_629r
    ==
    model_sha_before_629r
)

if not model_state_unchanged_629r:
    raise RuntimeError(
        "FD003 trained-model weights changed during FIT calibration execution."
    )

if bool(
    model_629r.training
):
    raise RuntimeError(
        "FD003 model was not restored to eval mode after FIT MC execution."
    )


# ======================================================================================
# 10) Freeze arrays + fingerprints
# ======================================================================================

for arr_629r in [
    FD003_FIT_DETERMINISTIC_PREDICTION_629R,
    FD003_FIT_MC_MATRIX_629R,
    FD003_FIT_MC_MEAN_629R,
    FD003_FIT_MC_STD_629R,
    FD003_PHASE_EDGES_629R,
    FD003_FIT_PHASE_ID_629R,
]:

    arr_629r.setflags(
        write=False
    )

FD003_FIT_DETERMINISTIC_PREDICTION_SHA256_629R = (
    _raw_sha256_629r(
        FD003_FIT_DETERMINISTIC_PREDICTION_629R
    )
)

FD003_FIT_MC_MATRIX_SHA256_629R = (
    _raw_sha256_629r(
        FD003_FIT_MC_MATRIX_629R
    )
)

FD003_FIT_MC_MEAN_SHA256_629R = (
    _raw_sha256_629r(
        FD003_FIT_MC_MEAN_629R
    )
)

FD003_FIT_MC_STD_SHA256_629R = (
    _raw_sha256_629r(
        FD003_FIT_MC_STD_629R
    )
)

FD003_PHASE_EDGES_SHA256_629R = (
    _raw_sha256_629r(
        FD003_PHASE_EDGES_629R
    )
)

FD003_FIT_PHASE_ID_SHA256_629R = (
    _raw_sha256_629r(
        FD003_FIT_PHASE_ID_629R
    )
)

FD003_PHASE_REFERENCE_SHA256_629R = {
    int(
        phase_629r
    ):
        _raw_sha256_629r(
            FD003_PHASE_REFERENCES_629R[
                phase_629r
            ]
        )
    for phase_629r in range(5)
}


# ======================================================================================
# 11) Calibration evidence contract
# ======================================================================================

FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_629R = {
    "step":
        "6.29R",

    "dataset":
        "FD003",

    "population":
        "all_17114_FIT_windows",

    "calibration_protocol_sha256":
        FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_SHA256_629Q,

    "trained_model_state_sha256":
        model_sha_after_629r,

    "canonical_mc_helper_signature":
        mc_helper_signature_629r,

    "canonical_mc_helper_bytecode_sha256":
        mc_helper_bytecode_sha256_629r,

    "deterministic_prediction_sha256":
        FD003_FIT_DETERMINISTIC_PREDICTION_SHA256_629R,

    "mc_matrix_sha256":
        FD003_FIT_MC_MATRIX_SHA256_629R,

    "mc_mean_sha256":
        FD003_FIT_MC_MEAN_SHA256_629R,

    "mc_std_sha256":
        FD003_FIT_MC_STD_SHA256_629R,

    "phase_edge_quantile_probabilities":
        list(
            FD003_PHASE_QUANTILE_PROBS_629Q
        ),

    "phase_edge_method":
        "linear",

    "phase_assignment":
        "np.digitize(right=True)",

    "phase_edges_sha256":
        FD003_PHASE_EDGES_SHA256_629R,

    "phase_counts":
        list(
            phase_counts_629r
        ),

    "phase_id_sha256":
        FD003_FIT_PHASE_ID_SHA256_629R,

    "phase_reference_lengths":
        {
            int(
                phase_629r
            ):
                int(
                    len(
                        FD003_PHASE_REFERENCES_629R[
                            phase_629r
                        ]
                    )
                )
            for phase_629r in range(5)
        },

    "phase_reference_sha256":
        dict(
            FD003_PHASE_REFERENCE_SHA256_629R
        ),

    "within_phase_reference":
        "sorted_FIT_MC_STD",

    "within_phase_mapping":
        "empirical_midrank_percentile",

    "fit_labels_used":
        False,

    "dev_labels_used":
        False,

    "dev_scoring_performed":
        False,

    "fd003_risk_created":
        False,

    "threshold_selected":
        False,

    "model_retrained":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629r = json.dumps(
    FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_629R,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_629R = (
    hashlib.sha256(
        contract_payload_629r
    ).hexdigest()
)

FD003_FIT_CALIBRATION_LOCKED_629R = True

FD003_MODEL_INFERENCE_PERFORMED_629R = True
FD003_MC_EXECUTION_PERFORMED_629R = True
FD003_FIT_LABELS_USED_629R = False
FD003_DEV_LABELS_USED_629R = False
FD003_DEV_SCORING_PERFORMED_629R = False
FD003_NEW_RISK_CREATED_629R = False
FD003_THRESHOLD_SELECTED_629R = False
FD003_MODEL_RETRAINED_629R = False
FD003_TEST_CONTENTS_READ_629R = False
FD003_RUL_CONTENTS_READ_629R = False

all_checks_passed_629r = True


# ======================================================================================
# 12) Output
# ======================================================================================

print("=" * 132)

print(
    "STEP 6.29R — FD003 FIT-ONLY DETERMINISTIC + MC "
    "CALIBRATION EXECUTION / PHASE-REFERENCE LOCK"
)

print("=" * 132)

print("Model inference performed      :", True)
print("MC-dropout execution performed :", True)
print("FIT labels used                :", False)
print("DEV labels used                :", False)
print("DEV scoring performed          :", False)
print("FD003 risk created             :", False)
print("Threshold selected             :", False)
print("Model retrained                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FIT EXECUTION CONTRACT ---")

print(
    "FIT windows                    :",
    len(
        FD003_FIT_DETERMINISTIC_PREDICTION_629R
    ),
)

print(
    "MC matrix shape                :",
    FD003_FIT_MC_MATRIX_629R.shape,
)

print(
    "MC passes / seed / ddof        :",
    (
        FD003_MC_PASSES_FOR_FIT_CALIBRATION_629Q,
        FD003_MC_SEED_FOR_FIT_CALIBRATION_629Q,
        FD003_MC_STD_DDOF_FOR_FIT_CALIBRATION_629Q,
    ),
)

print(
    "MC runtime seconds             :",
    FD003_FIT_MC_RUNTIME_SECONDS_629R,
)

print(
    "Model state unchanged          :",
    model_state_unchanged_629r,
)

print("\n--- FD003 PHASE CALIBRATION ---")

print(
    "Phase edges                    :",
    FD003_PHASE_EDGES_629R.tolist(),
)

print(
    "Phase counts                   :",
    phase_counts_629r,
)

print(
    "Reference lengths              :",
    [
        len(
            FD003_PHASE_REFERENCES_629R[
                phase_629r
            ]
        )
        for phase_629r in range(5)
    ],
)

print(
    "MC STD min / median / mean / max:",
    (
        float(
            FD003_FIT_MC_STD_629R.min()
        ),
        float(
            np.median(
                FD003_FIT_MC_STD_629R
            )
        ),
        float(
            FD003_FIT_MC_STD_629R.mean()
        ),
        float(
            FD003_FIT_MC_STD_629R.max()
        ),
    ),
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Deterministic FIT prediction SHA:",
    FD003_FIT_DETERMINISTIC_PREDICTION_SHA256_629R,
)

print(
    "FIT MC matrix SHA-256           :",
    FD003_FIT_MC_MATRIX_SHA256_629R,
)

print(
    "FIT MC STD SHA-256              :",
    FD003_FIT_MC_STD_SHA256_629R,
)

print(
    "Phase edges SHA-256             :",
    FD003_PHASE_EDGES_SHA256_629R,
)

print(
    "Phase-reference SHA-256         :",
    FD003_PHASE_REFERENCE_SHA256_629R,
)

print(
    "Calibration evidence SHA-256    :",
    FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_629R,
)

print(
    "\nPASS — the FD003 FIT-only uncertainty calibration execution "
    "is complete and the five phase-specific MC-STD references are frozen."
)

print(
    "No FIT/DEV labels, TEST, official RUL, risk threshold, or retuning "
    "were used."
)

print(
    "Next: apply the frozen FIT phase calibration to the already-locked "
    "clean DEV deterministic prediction + MC STD, then construct the "
    "clean-DEV G1 reference. Threshold selection still remains forbidden."
)

In [ ]:
# STEP 6.29S
# FD003 clean-DEV phase adjustment + G1 reference / right-ECDF risk lock
#
# REUSES already-locked clean DEV deterministic prediction + MC STD.
# NO model inference / NO MC execution / NO DEV labels / NO TEST / NO official RUL.
# NO threshold selection / NO retraining / NO architecture or policy changes.

import hashlib
import json
import numpy as np


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629r", False)):
    raise RuntimeError(
        "STEP 6.29R must pass before STEP 6.29S."
    )

required_629s = [
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_DEV_MC_STD_629M",
    "FD003_DEV_PREDICTION_SHA256_629K",
    "FD003_DEV_MC_STD_SHA256_629M",
    "FD003_PHASE_EDGES_629R",
    "FD003_PHASE_REFERENCES_629R",
    "FD003_PHASE_EDGES_SHA256_629R",
    "FD003_PHASE_REFERENCE_SHA256_629R",
    "FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_629R",
    "FD003_MC_DEV_CONTRACT_SHA256_629M",
]

missing_629s = [
    name
    for name in required_629s
    if name not in globals()
]

if missing_629s:
    raise RuntimeError(
        "Missing required FD003 G1-construction object(s): "
        + ", ".join(missing_629s)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629R", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29R."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629R", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29R."
    )

if bool(globals().get("FD003_G1_REFERENCE_LOCKED_629S", False)):
    raise RuntimeError(
        "STEP 6.29S has already locked the FD003 clean-DEV G1 reference."
    )


# ======================================================================================
# 2) Hash helper
# ======================================================================================

def _raw_sha256_629s(value):
    arr_629s = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629s.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29S hashing."
        )

    return hashlib.sha256(
        arr_629s.tobytes(order="C")
    ).hexdigest()


# ======================================================================================
# 3) Verify reused clean-DEV evidence has not changed
# ======================================================================================

dev_prediction_629s = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)

dev_mc_std_629s = np.asarray(
    FD003_DEV_MC_STD_629M,
    dtype=np.float64,
).reshape(-1)

if dev_prediction_629s.shape != (4706,):
    raise RuntimeError(
        "FD003 clean DEV deterministic prediction shape changed."
    )

if dev_mc_std_629s.shape != (4706,):
    raise RuntimeError(
        "FD003 clean DEV MC STD shape changed."
    )

if not np.isfinite(dev_prediction_629s).all():
    raise RuntimeError(
        "FD003 clean DEV deterministic prediction contains non-finite values."
    )

if not np.isfinite(dev_mc_std_629s).all():
    raise RuntimeError(
        "FD003 clean DEV MC STD contains non-finite values."
    )

dev_prediction_sha_629s = _raw_sha256_629s(
    dev_prediction_629s
)

dev_mc_std_sha_629s = _raw_sha256_629s(
    dev_mc_std_629s
)

if (
    dev_prediction_sha_629s
    !=
    FD003_DEV_PREDICTION_SHA256_629K
):
    raise RuntimeError(
        "FD003 clean DEV deterministic prediction fingerprint changed."
    )

if (
    dev_mc_std_sha_629s
    !=
    FD003_DEV_MC_STD_SHA256_629M
):
    raise RuntimeError(
        "FD003 clean DEV MC STD fingerprint changed."
    )


# ======================================================================================
# 4) Verify FIT phase-calibration references before reuse
# ======================================================================================

phase_edges_629s = np.asarray(
    FD003_PHASE_EDGES_629R,
    dtype=np.float64,
).reshape(-1)

if phase_edges_629s.shape != (4,):
    raise RuntimeError(
        "FD003 frozen phase-edge vector is not length 4."
    )

if (
    _raw_sha256_629s(
        phase_edges_629s
    )
    !=
    FD003_PHASE_EDGES_SHA256_629R
):
    raise RuntimeError(
        "FD003 frozen phase-edge fingerprint changed."
    )

phase_reference_hashes_now_629s = {}

for phase_629s in range(5):

    if phase_629s not in FD003_PHASE_REFERENCES_629R:
        raise RuntimeError(
            f"Missing frozen FD003 FIT phase reference {phase_629s}."
        )

    reference_629s = np.asarray(
        FD003_PHASE_REFERENCES_629R[
            phase_629s
        ],
        dtype=np.float64,
    ).reshape(-1)

    if len(reference_629s) <= 0:
        raise RuntimeError(
            f"FD003 FIT phase reference {phase_629s} is empty."
        )

    if not np.isfinite(reference_629s).all():
        raise RuntimeError(
            f"FD003 FIT phase reference {phase_629s} contains non-finite values."
        )

    if not bool(
        np.all(
            np.diff(
                reference_629s
            )
            >=
            0.0
        )
    ):
        raise RuntimeError(
            f"FD003 FIT phase reference {phase_629s} is not sorted."
        )

    hash_629s = _raw_sha256_629s(
        reference_629s
    )

    phase_reference_hashes_now_629s[
        phase_629s
    ] = hash_629s

    if (
        hash_629s
        !=
        FD003_PHASE_REFERENCE_SHA256_629R[
            phase_629s
        ]
    ):
        raise RuntimeError(
            f"FD003 FIT phase-reference {phase_629s} fingerprint changed."
        )


# ======================================================================================
# 5) FD003 phase-adjustment function
#
# Exact recovered semantics:
#   phase = np.digitize(prediction, FIT phase edges, right=True)
#   within phase:
#       adjusted = (left_rank + right_rank) / (2 * reference_length)
# ======================================================================================

def apply_fd003_phase_adjustment_629s(
    deterministic_prediction,
    mc_std,
):
    prediction_local_629s = np.asarray(
        deterministic_prediction,
        dtype=np.float64,
    ).reshape(-1)

    uncertainty_local_629s = np.asarray(
        mc_std,
        dtype=np.float64,
    ).reshape(-1)

    if (
        prediction_local_629s.shape
        !=
        uncertainty_local_629s.shape
    ):
        raise ValueError(
            "FD003 prediction and MC-STD shape mismatch."
        )

    phase_id_local_629s = np.digitize(
        prediction_local_629s,
        FD003_PHASE_EDGES_629R,
        right=True,
    )

    adjusted_local_629s = np.zeros(
        len(
            prediction_local_629s
        ),
        dtype=np.float64,
    )

    for phase_local_629s in np.unique(
        phase_id_local_629s
    ):
        phase_int_629s = int(
            phase_local_629s
        )

        if (
            phase_int_629s
            not in
            FD003_PHASE_REFERENCES_629R
        ):
            raise RuntimeError(
                f"Missing frozen FD003 FIT reference for phase {phase_int_629s}."
            )

        mask_local_629s = (
            phase_id_local_629s
            ==
            phase_local_629s
        )

        reference_local_629s = np.asarray(
            FD003_PHASE_REFERENCES_629R[
                phase_int_629s
            ],
            dtype=np.float64,
        ).reshape(-1)

        x_local_629s = uncertainty_local_629s[
            mask_local_629s
        ]

        left_local_629s = np.searchsorted(
            reference_local_629s,
            x_local_629s,
            side="left",
        )

        right_local_629s = np.searchsorted(
            reference_local_629s,
            x_local_629s,
            side="right",
        )

        adjusted_local_629s[
            mask_local_629s
        ] = (
            left_local_629s
            +
            right_local_629s
        ) / (
            2.0
            *
            len(
                reference_local_629s
            )
        )

    return adjusted_local_629s


# ======================================================================================
# 6) Apply frozen FIT calibration to clean DEV
# ======================================================================================

FD003_DEV_PHASE_ID_629S = np.digitize(
    dev_prediction_629s,
    phase_edges_629s,
    right=True,
).astype(
    np.int64
)

FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S = (
    apply_fd003_phase_adjustment_629s(
        dev_prediction_629s,
        dev_mc_std_629s,
    )
)

if (
    FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S.shape
    !=
    (4706,)
):
    raise RuntimeError(
        "FD003 DEV phase-adjusted uncertainty shape mismatch."
    )

if not np.isfinite(
    FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
).all():
    raise RuntimeError(
        "FD003 DEV phase-adjusted uncertainty contains non-finite values."
    )

if bool(
    (
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
        <
        0.0
    ).any()
    or
    (
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
        >
        1.0
    ).any()
):
    raise RuntimeError(
        "FD003 DEV phase-adjusted uncertainty is outside [0, 1]."
    )

dev_phase_counts_629s = [
    int(
        np.sum(
            FD003_DEV_PHASE_ID_629S
            ==
            phase_629s
        )
    )
    for phase_629s in range(5)
]

if sum(
    dev_phase_counts_629s
) != 4706:
    raise RuntimeError(
        "FD003 DEV phase counts do not cover all DEV windows."
    )


# ======================================================================================
# 7) Construct clean-DEV G1 reference
# ======================================================================================

FD003_G1_REFERENCE_629S = np.sort(
    np.asarray(
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S,
        dtype=np.float64,
    )
)

if (
    FD003_G1_REFERENCE_629S.shape
    !=
    (4706,)
):
    raise RuntimeError(
        "FD003 G1 reference shape mismatch."
    )

if not bool(
    np.all(
        np.diff(
            FD003_G1_REFERENCE_629S
        )
        >=
        0.0
    )
):
    raise RuntimeError(
        "FD003 G1 reference is not sorted."
    )


# ======================================================================================
# 8) Apply historical G1 right empirical CDF
# ======================================================================================

def apply_fd003_g1_right_ecdf_629s(
    adjusted_uncertainty,
):
    adjusted_local_629s = np.asarray(
        adjusted_uncertainty,
        dtype=np.float64,
    ).reshape(-1)

    if not np.isfinite(
        adjusted_local_629s
    ).all():
        raise ValueError(
            "FD003 G1 input contains non-finite values."
        )

    return (
        np.searchsorted(
            FD003_G1_REFERENCE_629S,
            adjusted_local_629s,
            side="right",
        )
        /
        float(
            len(
                FD003_G1_REFERENCE_629S
            )
        )
    ).astype(
        np.float64
    )


FD003_CLEAN_DEV_G1_RISK_629S = (
    apply_fd003_g1_right_ecdf_629s(
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
    )
)

if (
    FD003_CLEAN_DEV_G1_RISK_629S.shape
    !=
    (4706,)
):
    raise RuntimeError(
        "FD003 clean DEV G1 risk shape mismatch."
    )

if not np.isfinite(
    FD003_CLEAN_DEV_G1_RISK_629S
).all():
    raise RuntimeError(
        "FD003 clean DEV G1 risk contains non-finite values."
    )

if bool(
    (
        FD003_CLEAN_DEV_G1_RISK_629S
        <
        0.0
    ).any()
    or
    (
        FD003_CLEAN_DEV_G1_RISK_629S
        >
        1.0
    ).any()
):
    raise RuntimeError(
        "FD003 clean DEV G1 risk is outside [0, 1]."
    )


# ======================================================================================
# 9) Freeze arrays + fingerprints
# ======================================================================================

for arr_629s in [
    FD003_DEV_PHASE_ID_629S,
    FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S,
    FD003_G1_REFERENCE_629S,
    FD003_CLEAN_DEV_G1_RISK_629S,
]:
    arr_629s.setflags(
        write=False
    )

FD003_DEV_PHASE_ID_SHA256_629S = _raw_sha256_629s(
    FD003_DEV_PHASE_ID_629S
)

FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S = (
    _raw_sha256_629s(
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
    )
)

FD003_G1_REFERENCE_SHA256_629S = _raw_sha256_629s(
    FD003_G1_REFERENCE_629S
)

FD003_CLEAN_DEV_G1_RISK_SHA256_629S = _raw_sha256_629s(
    FD003_CLEAN_DEV_G1_RISK_629S
)


# ======================================================================================
# 10) G1 evidence contract
# ======================================================================================

FD003_G1_EVIDENCE_CONTRACT_629S = {
    "step":
        "6.29S",

    "dataset":
        "FD003",

    "phase_calibration_source":
        "frozen_FD003_FIT_only",

    "fit_calibration_evidence_contract_sha256":
        FD003_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_629R,

    "clean_dev_mc_contract_sha256":
        FD003_MC_DEV_CONTRACT_SHA256_629M,

    "clean_dev_population":
        "all_4706_DEV_windows",

    "phase_assignment":
        "np.digitize(prediction, frozen_FIT_edges, right=True)",

    "within_phase_mapping":
        "empirical_midrank_percentile_against_FIT_MC_STD_reference",

    "dev_phase_counts":
        list(
            dev_phase_counts_629s
        ),

    "adjusted_uncertainty_sha256":
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S,

    "g1_reference_source":
        "clean_FD003_DEV_windows",

    "g1_reference_size":
        int(
            len(
                FD003_G1_REFERENCE_629S
            )
        ),

    "g1_transform":
        "right_empirical_CDF",

    "g1_reference_sha256":
        FD003_G1_REFERENCE_SHA256_629S,

    "clean_dev_g1_risk_sha256":
        FD003_CLEAN_DEV_G1_RISK_SHA256_629S,

    "risk_direction":
        "larger_value_means_lower_reliability",

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "g2_reference_created":
        False,

    "final_selective_risk_created":
        False,

    "threshold_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

g1_contract_payload_629s = json.dumps(
    FD003_G1_EVIDENCE_CONTRACT_629S,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_G1_EVIDENCE_CONTRACT_SHA256_629S = hashlib.sha256(
    g1_contract_payload_629s
).hexdigest()

FD003_G1_REFERENCE_LOCKED_629S = True
FD003_G1_RISK_CREATED_629S = True
FD003_G2_REFERENCE_CREATED_629S = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629S = False

FD003_MODEL_INFERENCE_PERFORMED_629S = False
FD003_MC_EXECUTION_PERFORMED_629S = False
FD003_DEV_LABELS_USED_629S = False
FD003_DEV_SCORING_PERFORMED_629S = False
FD003_THRESHOLD_SELECTED_629S = False
FD003_TEST_CONTENTS_READ_629S = False
FD003_RUL_CONTENTS_READ_629S = False

all_checks_passed_629s = True


# ======================================================================================
# 11) Output
# ======================================================================================

print("=" * 132)

print(
    "STEP 6.29S — FD003 CLEAN-DEV PHASE ADJUSTMENT + "
    "G1 REFERENCE / RIGHT-ECDF RISK LOCK"
)

print("=" * 132)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("G1 risk created                :", True)
print("G2 reference created           :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- CLEAN DEV PHASE CALIBRATION ---")

print(
    "DEV windows                    :",
    len(
        dev_prediction_629s
    ),
)

print(
    "DEV phase counts               :",
    dev_phase_counts_629s,
)

print(
    "Adjusted uncertainty min/median/mean/max:",
    (
        float(
            FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S.min()
        ),
        float(
            np.median(
                FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S
            )
        ),
        float(
            FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S.mean()
        ),
        float(
            FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S.max()
        ),
    ),
)

print("\n--- G1 REFERENCE / RISK ---")

print(
    "G1 reference size              :",
    len(
        FD003_G1_REFERENCE_629S
    ),
)

print(
    "G1 reference sorted            :",
    True,
)

print(
    "G1 reference unique values     :",
    int(
        len(
            np.unique(
                FD003_G1_REFERENCE_629S
            )
        )
    ),
)

print(
    "Clean DEV G1 risk min/median/mean/max:",
    (
        float(
            FD003_CLEAN_DEV_G1_RISK_629S.min()
        ),
        float(
            np.median(
                FD003_CLEAN_DEV_G1_RISK_629S
            )
        ),
        float(
            FD003_CLEAN_DEV_G1_RISK_629S.mean()
        ),
        float(
            FD003_CLEAN_DEV_G1_RISK_629S.max()
        ),
    ),
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "DEV adjusted uncertainty SHA-256:",
    FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S,
)

print(
    "G1 reference SHA-256            :",
    FD003_G1_REFERENCE_SHA256_629S,
)

print(
    "Clean DEV G1 risk SHA-256       :",
    FD003_CLEAN_DEV_G1_RISK_SHA256_629S,
)

print(
    "G1 evidence contract SHA-256    :",
    FD003_G1_EVIDENCE_CONTRACT_SHA256_629S,
)

print(
    "\nPASS — the frozen FIT phase calibration has been applied "
    "to clean FD003 DEV, and the clean-DEV G1 reference/right-ECDF "
    "risk is now locked."
)

print(
    "No DEV labels, TEST, official RUL, threshold selection, "
    "model inference, or MC rerun occurred in this step."
)

print(
    "Next: construct the historical-methodology-compatible "
    "engine-balanced clean-DEV G2 reference from candidate endpoints, "
    "then freeze the final clean selective-risk transform. "
    "Threshold selection remains forbidden."
)

In [ ]:
# STEP 6.29T
# FD003 — historical G2 engine-balancing provenance verification
#
# READ-ONLY FORENSIC STEP.
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO FD003 G2 construction / NO threshold selection.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629s", False)):
    raise RuntimeError(
        "STEP 6.29S must pass before STEP 6.29T."
    )

required_629t = [
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "apply_fd002_g2_weighted_midrank_628hc0r2",
]

missing_629t = [
    name
    for name in required_629t
    if name not in globals()
]

if missing_629t:
    raise RuntimeError(
        "Missing required historical G2 object(s): "
        + ", ".join(missing_629t)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629S", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29S."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629S", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29S."
    )


# ======================================================================================
# 2) Helpers
# ======================================================================================

FORBIDDEN_629T = (
    "test",
    "rul",
    "label",
    "target",
    "error",
    "mae",
    "score",
    "scoring",
)

def _raw_sha256_629t(value):

    arr_629t = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629t.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29T hashing."
        )

    return hashlib.sha256(
        arr_629t.tobytes(order="C")
    ).hexdigest()


def _safe_value_629t(value):

    if value is None or isinstance(
        value,
        (
            str,
            bool,
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        if isinstance(value, np.integer):
            return int(value)

        if isinstance(value, np.floating):
            return float(value)

        return value

    if isinstance(value, np.ndarray):

        arr_629t = np.asarray(
            value
        )

        descriptor_629t = {
            "kind": "ndarray",
            "shape": tuple(
                int(x)
                for x in arr_629t.shape
            ),
            "dtype": str(
                arr_629t.dtype
            ),
        }

        if (
            not arr_629t.dtype.hasobject
            and np.issubdtype(
                arr_629t.dtype,
                np.number,
            )
        ):
            descriptor_629t["finite"] = bool(
                np.isfinite(
                    arr_629t
                ).all()
            )

            descriptor_629t["sha256"] = _raw_sha256_629t(
                arr_629t
            )

        return descriptor_629t

    if isinstance(value, dict):

        safe_dict_629t = {}

        for key_629t, item_629t in value.items():

            key_text_629t = str(
                key_629t
            )

            if any(
                token_629t in key_text_629t.lower()
                for token_629t in FORBIDDEN_629T
            ):
                continue

            safe_dict_629t[
                key_text_629t
            ] = _safe_value_629t(
                item_629t
            )

        return safe_dict_629t

    if isinstance(value, (list, tuple)):

        if (
            len(value) <= 100
            and all(
                item_629t is None
                or isinstance(
                    item_629t,
                    (
                        str,
                        bool,
                        int,
                        float,
                        np.integer,
                        np.floating,
                    ),
                )
                for item_629t in value
            )
        ):
            return [
                _safe_value_629t(
                    item_629t
                )
                for item_629t in value
            ]

        return {
            "kind": type(value).__name__,
            "length": int(
                len(value)
            ),
        }

    return {
        "kind": type(value).__name__
    }


# ======================================================================================
# 3) Historical G2 core integrity
# ======================================================================================

g2_values_629t = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

g2_weights_629t = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

if g2_values_629t.shape != g2_weights_629t.shape:
    raise RuntimeError(
        "Historical G2 values/weights shape mismatch."
    )

if g2_values_629t.shape != (8367,):
    raise RuntimeError(
        "Historical FD002 G2 reference size is not 8367."
    )

if not np.isfinite(g2_values_629t).all():
    raise RuntimeError(
        "Historical G2 values contain non-finite values."
    )

if not np.isfinite(g2_weights_629t).all():
    raise RuntimeError(
        "Historical G2 weights contain non-finite values."
    )

if bool(
    (
        g2_weights_629t
        <
        0.0
    ).any()
):
    raise RuntimeError(
        "Historical G2 weights contain negative values."
    )

g2_weight_sum_629t = float(
    g2_weights_629t.sum()
)

if not np.isclose(
    g2_weight_sum_629t,
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Historical G2 weights do not sum to 1."
    )


# ======================================================================================
# 4) Recover exact historical weighted-midrank implementation source
# ======================================================================================

g2_helper_629t = (
    apply_fd002_g2_weighted_midrank_628hc0r2
)

try:
    g2_helper_signature_629t = str(
        inspect.signature(
            g2_helper_629t
        )
    )
except (TypeError, ValueError):
    g2_helper_signature_629t = "<unavailable>"

try:
    g2_helper_source_629t = inspect.getsource(
        g2_helper_629t
    )

    g2_helper_source_available_629t = True

    g2_helper_source_sha256_629t = hashlib.sha256(
        g2_helper_source_629t.encode(
            "utf-8"
        )
    ).hexdigest()

except (OSError, TypeError):

    g2_helper_source_629t = None
    g2_helper_source_available_629t = False
    g2_helper_source_sha256_629t = None


# ======================================================================================
# 5) Narrow historical namespace inventory
# ======================================================================================

candidate_names_629t = []

for name_629t in sorted(
    globals().keys()
):

    lower_629t = name_629t.lower()

    historical_629t = (
        "628" in lower_629t
        or
        "fd002" in lower_629t
    )

    forbidden_629t = any(
        token_629t in lower_629t
        for token_629t in FORBIDDEN_629T
    )

    g2_related_629t = any(
        token_629t in lower_629t
        for token_629t in (
            "628fcr2",
            "g2_reference",
            "g2_weight",
            "weighted_midrank",
            "engine_balanc",
            "candidate_endpoint",
        )
    )

    dev_key_related_629t = (
        "dev" in lower_629t
        and any(
            token_629t in lower_629t
            for token_629t in (
                "key",
                "engine",
                "unit",
                "endpoint",
                "window",
            )
        )
    )

    if (
        historical_629t
        and not forbidden_629t
        and (
            g2_related_629t
            or dev_key_related_629t
        )
    ):
        candidate_names_629t.append(
            name_629t
        )


inventory_rows_629t = []
callable_sources_629t = {}
primitive_metadata_629t = {}
engine_vector_candidates_629t = {}

for name_629t in candidate_names_629t:

    obj_629t = globals()[
        name_629t
    ]

    row_629t = {
        "name": name_629t,
        "type": type(
            obj_629t
        ).__name__,
        "shape": None,
        "dtype": None,
        "callable": bool(
            callable(
                obj_629t
            )
        ),
        "notes": None,
    }

    if callable(
        obj_629t
    ):

        try:
            signature_629t = str(
                inspect.signature(
                    obj_629t
                )
            )
        except (TypeError, ValueError):
            signature_629t = "<unavailable>"

        try:
            source_629t = inspect.getsource(
                obj_629t
            )

            source_sha_629t = hashlib.sha256(
                source_629t.encode(
                    "utf-8"
                )
            ).hexdigest()

        except (OSError, TypeError):

            source_629t = None
            source_sha_629t = None

        callable_sources_629t[
            name_629t
        ] = {
            "signature":
                signature_629t,

            "source_sha256":
                source_sha_629t,

            "source":
                source_629t,
        }

        row_629t[
            "notes"
        ] = "source-only audit"

        inventory_rows_629t.append(
            row_629t
        )

        continue

    if isinstance(
        obj_629t,
        dict,
    ):

        primitive_metadata_629t[
            name_629t
        ] = _safe_value_629t(
            obj_629t
        )

        row_629t[
            "notes"
        ] = (
            f"dict with {len(obj_629t)} keys; "
            "safe metadata only"
        )

        inventory_rows_629t.append(
            row_629t
        )

        continue

    if isinstance(
        obj_629t,
        pd.DataFrame,
    ):

        row_629t[
            "shape"
        ] = tuple(
            int(x)
            for x in obj_629t.shape
        )

        row_629t[
            "notes"
        ] = (
            "columns="
            +
            repr(
                [
                    str(c)
                    for c in obj_629t.columns
                ]
            )
        )

        if len(
            obj_629t
        ) == 8367:

            for column_629t in obj_629t.columns:

                column_lower_629t = str(
                    column_629t
                ).lower()

                if (
                    "engine" in column_lower_629t
                    or
                    "unit" in column_lower_629t
                ):

                    candidate_629t = np.asarray(
                        obj_629t[
                            column_629t
                        ]
                    ).reshape(-1)

                    if (
                        len(
                            candidate_629t
                        )
                        ==
                        8367
                    ):
                        engine_vector_candidates_629t[
                            f"{name_629t}.{column_629t}"
                        ] = candidate_629t

        inventory_rows_629t.append(
            row_629t
        )

        continue

    try:
        arr_629t = np.asarray(
            obj_629t
        )
    except Exception:

        row_629t[
            "notes"
        ] = "non-array-like"

        inventory_rows_629t.append(
            row_629t
        )

        continue

    row_629t[
        "shape"
    ] = tuple(
        int(x)
        for x in arr_629t.shape
    )

    row_629t[
        "dtype"
    ] = str(
        arr_629t.dtype
    )

    lower_name_629t = name_629t.lower()

    if (
        arr_629t.ndim == 1
        and len(
            arr_629t
        ) == 8367
        and (
            "engine" in lower_name_629t
            or
            "unit" in lower_name_629t
        )
    ):
        engine_vector_candidates_629t[
            name_629t
        ] = arr_629t

        row_629t[
            "notes"
        ] = "8367-row explicit engine/unit vector candidate"

    elif (
        arr_629t.ndim == 2
        and arr_629t.shape[0] == 8367
        and arr_629t.shape[1] >= 1
        and (
            "key" in lower_name_629t
            or
            "window" in lower_name_629t
            or
            "endpoint" in lower_name_629t
        )
    ):
        first_column_629t = arr_629t[
            :,
            0
        ]

        engine_vector_candidates_629t[
            f"{name_629t}[:,0]_HYPOTHESIS"
        ] = first_column_629t

        row_629t[
            "notes"
        ] = (
            "8367-row key/window matrix; "
            "first-column engine hypothesis queued"
        )

    else:

        row_629t[
            "notes"
        ] = "structural audit only"

    inventory_rows_629t.append(
        row_629t
    )


FD003_HISTORICAL_G2_PROVENANCE_INVENTORY_629T = pd.DataFrame(
    inventory_rows_629t
)


# ======================================================================================
# 6) Engine-balanced weight formula probe
# ======================================================================================

weight_probe_rows_629t = []

for source_name_629t, engine_ids_raw_629t in (
    engine_vector_candidates_629t.items()
):

    engine_ids_629t = np.asarray(
        engine_ids_raw_629t
    ).reshape(-1)

    if len(
        engine_ids_629t
    ) != 8367:
        continue

    if engine_ids_629t.dtype.hasobject:

        try:
            engine_ids_629t = engine_ids_629t.astype(
                np.float64
            )
        except Exception:
            continue

    if np.issubdtype(
        engine_ids_629t.dtype,
        np.number,
    ):
        if not np.isfinite(
            engine_ids_629t
        ).all():
            continue

    unique_engines_629t, counts_629t = np.unique(
        engine_ids_629t,
        return_counts=True,
    )

    n_engines_629t = int(
        len(
            unique_engines_629t
        )
    )

    if n_engines_629t <= 1:
        continue

    count_map_629t = {
        engine_629t: int(
            count_629t
        )
        for engine_629t, count_629t
        in zip(
            unique_engines_629t,
            counts_629t,
        )
    }

    derived_weights_629t = np.asarray(
        [
            1.0
            /
            (
                n_engines_629t
                *
                count_map_629t[
                    engine_629t
                ]
            )
            for engine_629t
            in engine_ids_629t
        ],
        dtype=np.float64,
    )

    direct_exact_629t = bool(
        np.array_equal(
            derived_weights_629t,
            g2_weights_629t,
        )
    )

    direct_close_629t = bool(
        np.allclose(
            derived_weights_629t,
            g2_weights_629t,
            rtol=0.0,
            atol=1e-15,
        )
    )

    multiset_exact_629t = bool(
        np.array_equal(
            np.sort(
                derived_weights_629t
            ),
            np.sort(
                g2_weights_629t
            ),
        )
    )

    multiset_close_629t = bool(
        np.allclose(
            np.sort(
                derived_weights_629t
            ),
            np.sort(
                g2_weights_629t
            ),
            rtol=0.0,
            atol=1e-15,
        )
    )

    engine_mass_values_629t = np.asarray(
        [
            derived_weights_629t[
                engine_ids_629t
                ==
                engine_629t
            ].sum()
            for engine_629t
            in unique_engines_629t
        ],
        dtype=np.float64,
    )

    weight_probe_rows_629t.append(
        {
            "source":
                source_name_629t,

            "n_engines":
                n_engines_629t,

            "min_windows_per_engine":
                int(
                    counts_629t.min()
                ),

            "max_windows_per_engine":
                int(
                    counts_629t.max()
                ),

            "derived_weight_sum":
                float(
                    derived_weights_629t.sum()
                ),

            "equal_engine_mass":
                bool(
                    np.allclose(
                        engine_mass_values_629t,
                        np.full(
                            n_engines_629t,
                            1.0
                            /
                            n_engines_629t,
                            dtype=np.float64,
                        ),
                        rtol=0.0,
                        atol=1e-15,
                    )
                ),

            "direct_exact_historical_weights":
                direct_exact_629t,

            "direct_close_historical_weights":
                direct_close_629t,

            "multiset_exact_historical_weights":
                multiset_exact_629t,

            "multiset_close_historical_weights":
                multiset_close_629t,

            "derived_weight_sha256":
                _raw_sha256_629t(
                    derived_weights_629t
                ),
        }
    )


FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T = pd.DataFrame(
    weight_probe_rows_629t
)


# ======================================================================================
# 7) Conservative resolution
# ======================================================================================

if len(
    FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T
) > 0:

    exact_formula_rows_629t = (
        FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T[
            FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T[
                "direct_exact_historical_weights"
            ]
            |
            FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T[
                "direct_close_historical_weights"
            ]
        ]
    )

else:

    exact_formula_rows_629t = pd.DataFrame()


FD003_HISTORICAL_G2_ENGINE_BALANCE_RESOLVED_629T = bool(
    len(
        exact_formula_rows_629t
    )
    >=
    1
)


# ======================================================================================
# 8) Audit fingerprint
# ======================================================================================

audit_payload_629t = json.dumps(
    {
        "historical_g2_size":
            int(
                len(
                    g2_values_629t
                )
            ),

        "historical_g2_values_sha256":
            _raw_sha256_629t(
                g2_values_629t
            ),

        "historical_g2_weights_sha256":
            _raw_sha256_629t(
                g2_weights_629t
            ),

        "historical_g2_weight_sum":
            g2_weight_sum_629t,

        "weighted_midrank_source_sha256":
            g2_helper_source_sha256_629t,

        "candidate_names":
            candidate_names_629t,

        "safe_metadata_names":
            sorted(
                primitive_metadata_629t.keys()
            ),

        "weight_formula_probe":
            weight_probe_rows_629t,

        "engine_balance_resolved":
            FD003_HISTORICAL_G2_ENGINE_BALANCE_RESOLVED_629T,

        "fd003_g2_created":
            False,

        "dev_labels_used":
            False,

        "dev_error_scoring_performed":
            False,

        "model_inference_performed":
            False,

        "mc_execution_performed":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_HISTORICAL_G2_PROVENANCE_AUDIT_SHA256_629T = (
    hashlib.sha256(
        audit_payload_629t
    ).hexdigest()
)


FD003_G2_REFERENCE_CREATED_629T = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629T = False
FD003_DEV_LABELS_USED_629T = False
FD003_DEV_SCORING_PERFORMED_629T = False
FD003_MODEL_INFERENCE_PERFORMED_629T = False
FD003_MC_EXECUTION_PERFORMED_629T = False
FD003_THRESHOLD_SELECTED_629T = False
FD003_TEST_CONTENTS_READ_629T = False
FD003_RUL_CONTENTS_READ_629T = False

all_checks_passed_629t = True


# ======================================================================================
# 9) Output
# ======================================================================================

print("=" * 132)

print(
    "STEP 6.29T — FD003 HISTORICAL G2 ENGINE-BALANCING "
    "PROVENANCE VERIFICATION"
)

print("=" * 132)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- HISTORICAL G2 CORE ---")

print(
    "Reference size                 :",
    len(
        g2_values_629t
    ),
)

print(
    "Weight sum                     :",
    g2_weight_sum_629t,
)

print(
    "Unique G2 values               :",
    int(
        len(
            np.unique(
                g2_values_629t
            )
        )
    ),
)

print(
    "Unique G2 weights              :",
    int(
        len(
            np.unique(
                g2_weights_629t
            )
        )
    ),
)

print(
    "G2 values SHA-256              :",
    _raw_sha256_629t(
        g2_values_629t
    ),
)

print(
    "G2 weights SHA-256             :",
    _raw_sha256_629t(
        g2_weights_629t
    ),
)

print("\n--- WEIGHTED-MIDRANK IMPLEMENTATION ---")

print(
    "Signature                      :",
    g2_helper_signature_629t,
)

print(
    "Source available               :",
    g2_helper_source_available_629t,
)

print(
    "Source SHA-256                 :",
    g2_helper_source_sha256_629t,
)

if g2_helper_source_available_629t:
    print("\nSOURCE:\n")
    print(
        g2_helper_source_629t
    )

print("\n--- NARROW HISTORICAL INVENTORY ---")

if len(
    FD003_HISTORICAL_G2_PROVENANCE_INVENTORY_629T
) == 0:

    print(
        "<no additional historical G2/DEV-key objects found>"
    )

else:

    print(
        FD003_HISTORICAL_G2_PROVENANCE_INVENTORY_629T.to_string(
            index=False
        )
    )

if primitive_metadata_629t:

    print("\n--- SAFE HISTORICAL METADATA ---")

    for name_629t in sorted(
        primitive_metadata_629t.keys()
    ):

        print(
            f"\n{name_629t}"
        )

        print(
            json.dumps(
                primitive_metadata_629t[
                    name_629t
                ],
                indent=2,
                sort_keys=True,
                default=str,
            )
        )

print("\n--- ENGINE-BALANCE FORMULA PROBE ---")

if len(
    FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T
) == 0:

    print(
        "<no usable historical engine-membership vector recovered>"
    )

else:

    print(
        FD003_HISTORICAL_G2_WEIGHT_FORMULA_PROBE_629T.to_string(
            index=False
        )
    )

print("\n--- RESOLUTION ---")

print(
    "Exact/close equal-engine-mass formula recovered :",
    FD003_HISTORICAL_G2_ENGINE_BALANCE_RESOLVED_629T,
)

print(
    "Audit SHA-256                                :",
    FD003_HISTORICAL_G2_PROVENANCE_AUDIT_SHA256_629T,
)

print(
    "\nPASS — historical G2 construction provenance was audited only. "
    "No FD003 G2 reference or final selective risk was created."
)

print(
    "Next action depends on whether the engine-balanced weighting formula "
    "is exactly recovered. Threshold selection remains forbidden."
)

In [ ]:
# STEP 6.29T-R1
# FD003 — exact historical G2 value/weight pairing + ordering recovery
#
# READ-ONLY FORENSIC STEP.
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO FD003 G2 construction / NO threshold selection.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629t", False)):
    raise RuntimeError(
        "STEP 6.29T must pass before STEP 6.29T-R1."
    )

required_629tr1 = [
    "dev_g1_628fcr2",
    "dev_keys_628fcr2",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "sort_idx_628fcr2",
    "inverse_628fcr2",
    "weights_628fcr2",
    "engine_count_map_628fcr2",
    "n_engines_628fcr2",
    "apply_engine_balanced_g2_628fcr2",
]

missing_629tr1 = [
    name
    for name in required_629tr1
    if name not in globals()
]

if missing_629tr1:
    raise RuntimeError(
        "Missing required historical G2 provenance object(s): "
        + ", ".join(missing_629tr1)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629T", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29T."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629T", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29T."
    )


# ======================================================================================
# 2) Helpers
# ======================================================================================

def _raw_sha256_629tr1(value):

    arr_629tr1 = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629tr1.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29T-R1 hashing."
        )

    return hashlib.sha256(
        arr_629tr1.tobytes(order="C")
    ).hexdigest()


def _is_permutation_629tr1(index_array, n):

    idx_629tr1 = np.asarray(
        index_array,
        dtype=np.int64,
    ).reshape(-1)

    return bool(
        idx_629tr1.shape == (n,)
        and
        np.array_equal(
            np.sort(idx_629tr1),
            np.arange(
                n,
                dtype=np.int64,
            ),
        )
    )


# ======================================================================================
# 3) Strict-whitelist historical arrays
# ======================================================================================

dev_g1_629tr1 = np.asarray(
    dev_g1_628fcr2,
    dtype=np.float64,
).reshape(-1)

historical_values_629tr1 = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

historical_weights_629tr1 = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

historical_working_weights_629tr1 = np.asarray(
    weights_628fcr2,
    dtype=np.float64,
).reshape(-1)

sort_idx_629tr1 = np.asarray(
    sort_idx_628fcr2,
    dtype=np.int64,
).reshape(-1)

inverse_629tr1 = np.asarray(
    inverse_628fcr2,
    dtype=np.int64,
).reshape(-1)

if not isinstance(
    dev_keys_628fcr2,
    pd.DataFrame,
):
    raise RuntimeError(
        "Historical dev_keys_628fcr2 is not a DataFrame."
    )

if "unit_id" not in dev_keys_628fcr2.columns:
    raise RuntimeError(
        "Historical dev_keys_628fcr2 lacks unit_id."
    )

engine_ids_629tr1 = np.asarray(
    dev_keys_628fcr2[
        "unit_id"
    ]
).reshape(-1)

n_629tr1 = 8367

for name_629tr1, arr_629tr1 in [
    ("dev_g1", dev_g1_629tr1),
    ("historical_values", historical_values_629tr1),
    ("historical_weights", historical_weights_629tr1),
    ("working_weights", historical_working_weights_629tr1),
    ("sort_idx", sort_idx_629tr1),
    ("inverse", inverse_629tr1),
    ("engine_ids", engine_ids_629tr1),
]:

    if arr_629tr1.shape != (n_629tr1,):
        raise RuntimeError(
            f"{name_629tr1} historical shape is not ({n_629tr1},)."
        )

for name_629tr1, arr_629tr1 in [
    ("dev_g1", dev_g1_629tr1),
    ("historical_values", historical_values_629tr1),
    ("historical_weights", historical_weights_629tr1),
    ("working_weights", historical_working_weights_629tr1),
]:

    if not np.isfinite(
        arr_629tr1
    ).all():
        raise RuntimeError(
            f"{name_629tr1} contains non-finite values."
        )

if not _is_permutation_629tr1(
    sort_idx_629tr1,
    n_629tr1,
):
    raise RuntimeError(
        "Historical sort_idx_628fcr2 is not a permutation."
    )

if not _is_permutation_629tr1(
    inverse_629tr1,
    n_629tr1,
):
    raise RuntimeError(
        "Historical inverse_628fcr2 is not a permutation."
    )


# ======================================================================================
# 4) Re-derive equal-engine-mass raw weights
# ======================================================================================

n_engines_629tr1 = int(
    n_engines_628fcr2
)

if n_engines_629tr1 != 52:
    raise RuntimeError(
        "Historical FD002 G2 engine count is not 52."
    )

engine_count_map_local_629tr1 = {
    key: int(value)
    for key, value
    in engine_count_map_628fcr2.items()
}

if len(
    engine_count_map_local_629tr1
) != n_engines_629tr1:
    raise RuntimeError(
        "Historical engine-count map size does not equal n_engines."
    )

derived_raw_weights_629tr1 = np.asarray(
    [
        1.0
        /
        (
            n_engines_629tr1
            *
            engine_count_map_local_629tr1[
                engine_id_629tr1
            ]
        )
        for engine_id_629tr1
        in engine_ids_629tr1
    ],
    dtype=np.float64,
)

if not np.isclose(
    derived_raw_weights_629tr1.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Re-derived equal-engine-mass weights do not sum to 1."
    )


# ======================================================================================
# 5) Recover exact ordering semantics
# ======================================================================================

recomputed_mergesort_629tr1 = np.argsort(
    dev_g1_629tr1,
    kind="mergesort",
)

sort_idx_exact_mergesort_629tr1 = bool(
    np.array_equal(
        sort_idx_629tr1,
        recomputed_mergesort_629tr1,
    )
)

inverse_exact_629tr1 = bool(
    np.array_equal(
        inverse_629tr1[
            sort_idx_629tr1
        ],
        np.arange(
            n_629tr1,
            dtype=np.int64,
        ),
    )
    and
    np.array_equal(
        sort_idx_629tr1[
            inverse_629tr1
        ],
        np.arange(
            n_629tr1,
            dtype=np.int64,
        ),
    )
)

pathways_629tr1 = {
    "raw_dev_order": (
        dev_g1_629tr1,
        derived_raw_weights_629tr1,
    ),

    "historical_sort_idx": (
        dev_g1_629tr1[
            sort_idx_629tr1
        ],
        derived_raw_weights_629tr1[
            sort_idx_629tr1
        ],
    ),

    "recomputed_mergesort": (
        dev_g1_629tr1[
            recomputed_mergesort_629tr1
        ],
        derived_raw_weights_629tr1[
            recomputed_mergesort_629tr1
        ],
    ),

    "historical_inverse": (
        dev_g1_629tr1[
            inverse_629tr1
        ],
        derived_raw_weights_629tr1[
            inverse_629tr1
        ],
    ),
}

pairing_rows_629tr1 = []

for pathway_name_629tr1, (
    values_candidate_629tr1,
    weights_candidate_629tr1,
) in pathways_629tr1.items():

    pairing_rows_629tr1.append(
        {
            "pathway":
                pathway_name_629tr1,

            "values_exact":
                bool(
                    np.array_equal(
                        values_candidate_629tr1,
                        historical_values_629tr1,
                    )
                ),

            "weights_exact":
                bool(
                    np.array_equal(
                        weights_candidate_629tr1,
                        historical_weights_629tr1,
                    )
                ),

            "values_close_1e-15":
                bool(
                    np.allclose(
                        values_candidate_629tr1,
                        historical_values_629tr1,
                        rtol=0.0,
                        atol=1e-15,
                    )
                ),

            "weights_close_1e-15":
                bool(
                    np.allclose(
                        weights_candidate_629tr1,
                        historical_weights_629tr1,
                        rtol=0.0,
                        atol=1e-15,
                    )
                ),

            "candidate_values_sha256":
                _raw_sha256_629tr1(
                    values_candidate_629tr1
                ),

            "candidate_weights_sha256":
                _raw_sha256_629tr1(
                    weights_candidate_629tr1
                ),
        }
    )

FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1 = pd.DataFrame(
    pairing_rows_629tr1
)


# ======================================================================================
# 6) Check surviving historical working-weight object
# ======================================================================================

working_weight_checks_629tr1 = {
    "working_weights_equal_derived_raw":
        bool(
            np.array_equal(
                historical_working_weights_629tr1,
                derived_raw_weights_629tr1,
            )
        ),

    "working_weights_equal_derived_sorted":
        bool(
            np.array_equal(
                historical_working_weights_629tr1,
                derived_raw_weights_629tr1[
                    sort_idx_629tr1
                ],
            )
        ),

    "historical_reference_weights_equal_working":
        bool(
            np.array_equal(
                historical_weights_629tr1,
                historical_working_weights_629tr1,
            )
        ),

    "historical_reference_weights_equal_working_sorted":
        bool(
            np.array_equal(
                historical_weights_629tr1,
                historical_working_weights_629tr1[
                    sort_idx_629tr1
                ],
            )
        ),
}


# ======================================================================================
# 7) Recover exact historical construction-helper source
# ======================================================================================

construction_helper_629tr1 = (
    apply_engine_balanced_g2_628fcr2
)

if not callable(
    construction_helper_629tr1
):
    raise RuntimeError(
        "Historical engine-balanced G2 helper is not callable."
    )

try:

    construction_helper_signature_629tr1 = str(
        inspect.signature(
            construction_helper_629tr1
        )
    )

except (TypeError, ValueError):

    construction_helper_signature_629tr1 = "<unavailable>"

try:

    construction_helper_source_629tr1 = inspect.getsource(
        construction_helper_629tr1
    )

    construction_helper_source_available_629tr1 = True

    construction_helper_source_sha256_629tr1 = hashlib.sha256(
        construction_helper_source_629tr1.encode(
            "utf-8"
        )
    ).hexdigest()

except (OSError, TypeError):

    construction_helper_source_629tr1 = None
    construction_helper_source_available_629tr1 = False
    construction_helper_source_sha256_629tr1 = None


# ======================================================================================
# 8) Conservative resolution
# ======================================================================================

paired_exact_rows_629tr1 = (
    FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
        FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
            "values_exact"
        ]
        &
        FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
            "weights_exact"
        ]
    ]
)

paired_close_rows_629tr1 = (
    FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
        FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
            "values_close_1e-15"
        ]
        &
        FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1[
            "weights_close_1e-15"
        ]
    ]
)

FD003_HISTORICAL_G2_PAIRING_RESOLVED_629TR1 = bool(
    len(
        paired_exact_rows_629tr1
    )
    >=
    1
    or
    len(
        paired_close_rows_629tr1
    )
    >=
    1
)


# ======================================================================================
# 9) Audit fingerprint
# ======================================================================================

audit_payload_629tr1 = json.dumps(
    {
        "sort_idx_exact_mergesort":
            sort_idx_exact_mergesort_629tr1,

        "inverse_exact":
            inverse_exact_629tr1,

        "working_weight_checks":
            working_weight_checks_629tr1,

        "pairing_rows":
            pairing_rows_629tr1,

        "construction_helper_signature":
            construction_helper_signature_629tr1,

        "construction_helper_source_sha256":
            construction_helper_source_sha256_629tr1,

        "historical_g2_pairing_resolved":
            FD003_HISTORICAL_G2_PAIRING_RESOLVED_629TR1,

        "fd003_g2_created":
            False,

        "final_selective_risk_created":
            False,

        "dev_labels_used":
            False,

        "dev_error_scoring_performed":
            False,

        "model_inference_performed":
            False,

        "mc_execution_performed":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_HISTORICAL_G2_PAIRING_AUDIT_SHA256_629TR1 = (
    hashlib.sha256(
        audit_payload_629tr1
    ).hexdigest()
)


FD003_G2_REFERENCE_CREATED_629TR1 = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629TR1 = False
FD003_DEV_LABELS_USED_629TR1 = False
FD003_DEV_SCORING_PERFORMED_629TR1 = False
FD003_MODEL_INFERENCE_PERFORMED_629TR1 = False
FD003_MC_EXECUTION_PERFORMED_629TR1 = False
FD003_THRESHOLD_SELECTED_629TR1 = False
FD003_TEST_CONTENTS_READ_629TR1 = False
FD003_RUL_CONTENTS_READ_629TR1 = False

all_checks_passed_629tr1 = True


# ======================================================================================
# 10) Output
# ======================================================================================

print("=" * 134)

print(
    "STEP 6.29T-R1 — FD003 HISTORICAL G2 VALUE/WEIGHT "
    "PAIRING + ORDERING RECOVERY"
)

print("=" * 134)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- HISTORICAL ORDERING OBJECTS ---")

print(
    "sort_idx exact stable argsort(dev_g1) :",
    sort_idx_exact_mergesort_629tr1,
)

print(
    "inverse is exact inverse permutation  :",
    inverse_exact_629tr1,
)

print(
    "Historical values sorted nondecreasing:",
    bool(
        np.all(
            np.diff(
                historical_values_629tr1
            )
            >=
            0.0
        )
    ),
)

print("\n--- WORKING WEIGHT CHECKS ---")

for key_629tr1, value_629tr1 in (
    working_weight_checks_629tr1.items()
):
    print(
        f"{key_629tr1:48s}:",
        value_629tr1,
    )

print("\n--- VALUE/WEIGHT PAIRING PATHWAYS ---")

print(
    FD003_HISTORICAL_G2_PAIRING_AUDIT_629TR1.to_string(
        index=False
    )
)

print("\n--- HISTORICAL CONSTRUCTION HELPER ---")

print(
    "Signature                      :",
    construction_helper_signature_629tr1,
)

print(
    "Source available               :",
    construction_helper_source_available_629tr1,
)

print(
    "Source SHA-256                 :",
    construction_helper_source_sha256_629tr1,
)

if construction_helper_source_available_629tr1:

    print("\nSOURCE:\n")

    print(
        construction_helper_source_629tr1
    )

print("\n--- RESOLUTION ---")

print(
    "Exact/close value-weight pairing recovered :",
    FD003_HISTORICAL_G2_PAIRING_RESOLVED_629TR1,
)

print(
    "Audit SHA-256                              :",
    FD003_HISTORICAL_G2_PAIRING_AUDIT_SHA256_629TR1,
)

print(
    "\nPASS — this step only resolves the historical pairing/order "
    "of clean-DEV G1 values and equal-engine-mass weights."
)

print(
    "No FD003 G2 reference, final selective risk, or threshold "
    "has been created."
)

In [ ]:
# STEP 6.29T-R1-CORRECTED
# FD003 — historical G2 weighted-reference pairing / grouping recovery
#
# READ-ONLY FORENSIC STEP.
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO FD003 G2 construction / NO threshold selection.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629t", False)):
    raise RuntimeError(
        "STEP 6.29T must pass before STEP 6.29T-R1-CORRECTED."
    )

required_629tr1c = [
    "dev_g1_628fcr2",
    "dev_keys_628fcr2",
    "G2_REFERENCE_VALUES_FD002_628FCR2",
    "G2_REFERENCE_WEIGHTS_FD002_628FCR2",
    "sort_idx_628fcr2",
    "inverse_628fcr2",
    "unique_values_628fcr2",
    "unique_weights_628fcr2",
    "weights_628fcr2",
    "engine_count_map_628fcr2",
    "n_engines_628fcr2",
    "apply_engine_balanced_g2_628fcr2",
]

missing_629tr1c = [
    name
    for name in required_629tr1c
    if name not in globals()
]

if missing_629tr1c:
    raise RuntimeError(
        "Missing required historical G2 provenance object(s): "
        + ", ".join(missing_629tr1c)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629T", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29T."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629T", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29T."
    )


# ======================================================================================
# 2) Helpers
# ======================================================================================

def _raw_sha256_629tr1c(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29T-R1-CORRECTED hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _is_permutation_629tr1c(index_array, n):

    idx = np.asarray(
        index_array,
        dtype=np.int64,
    ).reshape(-1)

    return bool(
        idx.shape == (n,)
        and
        np.array_equal(
            np.sort(idx),
            np.arange(
                n,
                dtype=np.int64,
            ),
        )
    )


def _group_weight_mass_629tr1c(values, weights):

    values = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    weights = np.asarray(
        weights,
        dtype=np.float64,
    ).reshape(-1)

    if values.shape != weights.shape:
        raise ValueError(
            "Values/weights shape mismatch."
        )

    order = np.argsort(
        values,
        kind="mergesort",
    )

    values_sorted = values[
        order
    ]

    weights_sorted = weights[
        order
    ]

    unique_values, inverse = np.unique(
        values_sorted,
        return_inverse=True,
    )

    grouped_weights = np.bincount(
        inverse,
        weights=weights_sorted,
        minlength=len(unique_values),
    ).astype(
        np.float64
    )

    return (
        unique_values.astype(
            np.float64
        ),
        grouped_weights,
        order.astype(
            np.int64
        ),
        inverse.astype(
            np.int64
        ),
    )


# ======================================================================================
# 3) Strict-whitelist historical arrays
# ======================================================================================

dev_g1 = np.asarray(
    dev_g1_628fcr2,
    dtype=np.float64,
).reshape(-1)

historical_values = np.asarray(
    G2_REFERENCE_VALUES_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

historical_weights = np.asarray(
    G2_REFERENCE_WEIGHTS_FD002_628FCR2,
    dtype=np.float64,
).reshape(-1)

working_weights = np.asarray(
    weights_628fcr2,
    dtype=np.float64,
).reshape(-1)

sort_idx = np.asarray(
    sort_idx_628fcr2,
    dtype=np.int64,
).reshape(-1)

inverse = np.asarray(
    inverse_628fcr2,
    dtype=np.int64,
).reshape(-1)

stored_unique_values = np.asarray(
    unique_values_628fcr2,
    dtype=np.float64,
).reshape(-1)

stored_unique_weights = np.asarray(
    unique_weights_628fcr2,
    dtype=np.float64,
).reshape(-1)

if not isinstance(
    dev_keys_628fcr2,
    pd.DataFrame,
):
    raise RuntimeError(
        "Historical dev_keys_628fcr2 is not a DataFrame."
    )

if "unit_id" not in dev_keys_628fcr2.columns:
    raise RuntimeError(
        "Historical dev_keys_628fcr2 lacks unit_id."
    )

engine_ids = np.asarray(
    dev_keys_628fcr2[
        "unit_id"
    ]
).reshape(-1)

n = 8367

for name, arr in [
    ("dev_g1", dev_g1),
    ("historical_values", historical_values),
    ("historical_weights", historical_weights),
    ("working_weights", working_weights),
    ("sort_idx", sort_idx),
    ("inverse", inverse),
    ("engine_ids", engine_ids),
]:

    if arr.shape != (n,):
        raise RuntimeError(
            f"{name} historical shape is not ({n},)."
        )

for name, arr in [
    ("dev_g1", dev_g1),
    ("historical_values", historical_values),
    ("historical_weights", historical_weights),
    ("working_weights", working_weights),
    ("stored_unique_values", stored_unique_values),
    ("stored_unique_weights", stored_unique_weights),
]:

    if not np.isfinite(
        arr
    ).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )

if not _is_permutation_629tr1c(
    sort_idx,
    n,
):
    raise RuntimeError(
        "Historical sort_idx_628fcr2 is not a permutation."
    )

# IMPORTANT CORRECTION:
# inverse_628fcr2 is NOT assumed to be a permutation.
# It is tested as a possible np.unique(..., return_inverse=True) group-index vector.

inverse_range_valid = bool(
    inverse.min() >= 0
    and
    inverse.max() < len(
        stored_unique_values
    )
)


# ======================================================================================
# 4) Re-derive equal-engine-mass raw weights
# ======================================================================================

n_engines = int(
    n_engines_628fcr2
)

if n_engines != 52:
    raise RuntimeError(
        "Historical FD002 G2 engine count is not 52."
    )

engine_count_map = {
    key: int(value)
    for key, value
    in engine_count_map_628fcr2.items()
}

if len(
    engine_count_map
) != n_engines:
    raise RuntimeError(
        "Historical engine-count map size does not equal n_engines."
    )

derived_raw_weights = np.asarray(
    [
        1.0
        /
        (
            n_engines
            *
            engine_count_map[
                engine_id
            ]
        )
        for engine_id
        in engine_ids
    ],
    dtype=np.float64,
)

if not np.isclose(
    derived_raw_weights.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Re-derived equal-engine-mass weights do not sum to 1."
    )


# ======================================================================================
# 5) Test row-order pathways
# ======================================================================================

recomputed_mergesort = np.argsort(
    dev_g1,
    kind="mergesort",
)

sort_idx_exact_mergesort = bool(
    np.array_equal(
        sort_idx,
        recomputed_mergesort,
    )
)

pathways = {
    "raw_dev_order": (
        dev_g1,
        derived_raw_weights,
    ),

    "historical_sort_idx": (
        dev_g1[
            sort_idx
        ],
        derived_raw_weights[
            sort_idx
        ],
    ),

    "recomputed_mergesort": (
        dev_g1[
            recomputed_mergesort
        ],
        derived_raw_weights[
            recomputed_mergesort
        ],
    ),
}

pairing_rows = []

for pathway_name, (
    values_candidate,
    weights_candidate,
) in pathways.items():

    pairing_rows.append(
        {
            "pathway":
                pathway_name,

            "values_exact":
                bool(
                    np.array_equal(
                        values_candidate,
                        historical_values,
                    )
                ),

            "weights_exact":
                bool(
                    np.array_equal(
                        weights_candidate,
                        historical_weights,
                    )
                ),

            "values_close_1e-15":
                bool(
                    np.allclose(
                        values_candidate,
                        historical_values,
                        rtol=0.0,
                        atol=1e-15,
                    )
                ),

            "weights_close_1e-15":
                bool(
                    np.allclose(
                        weights_candidate,
                        historical_weights,
                        rtol=0.0,
                        atol=1e-15,
                    )
                ),

            "candidate_values_sha256":
                _raw_sha256_629tr1c(
                    values_candidate
                ),

            "candidate_weights_sha256":
                _raw_sha256_629tr1c(
                    weights_candidate
                ),
        }
    )

FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C = pd.DataFrame(
    pairing_rows
)


# ======================================================================================
# 6) Recover scientifically relevant weighted empirical distribution
# ======================================================================================

(
    hist_unique_values,
    hist_grouped_weights,
    hist_group_sort_idx,
    hist_group_inverse,
) = _group_weight_mass_629tr1c(
    historical_values,
    historical_weights,
)

(
    candidate_unique_values,
    candidate_grouped_weights,
    candidate_group_sort_idx,
    candidate_group_inverse,
) = _group_weight_mass_629tr1c(
    dev_g1,
    derived_raw_weights,
)

group_unique_values_exact = bool(
    np.array_equal(
        candidate_unique_values,
        hist_unique_values,
    )
)

group_weights_exact = bool(
    np.array_equal(
        candidate_grouped_weights,
        hist_grouped_weights,
    )
)

group_unique_values_close = bool(
    np.allclose(
        candidate_unique_values,
        hist_unique_values,
        rtol=0.0,
        atol=1e-15,
    )
)

group_weights_close = bool(
    np.allclose(
        candidate_grouped_weights,
        hist_grouped_weights,
        rtol=0.0,
        atol=1e-15,
    )
)

stored_unique_values_match_hist = bool(
    np.array_equal(
        stored_unique_values,
        hist_unique_values,
    )
)

stored_unique_weights_match_hist = bool(
    np.array_equal(
        stored_unique_weights,
        hist_grouped_weights,
    )
)

stored_unique_values_match_candidate = bool(
    np.array_equal(
        stored_unique_values,
        candidate_unique_values,
    )
)

stored_unique_weights_match_candidate = bool(
    np.array_equal(
        stored_unique_weights,
        candidate_grouped_weights,
    )
)


# ======================================================================================
# 7) Determine actual semantics of inverse_628fcr2
# ======================================================================================

inverse_reconstruction_raw = False
inverse_reconstruction_sorted = False

if inverse_range_valid:

    reconstructed_from_inverse = stored_unique_values[
        inverse
    ]

    inverse_reconstruction_raw = bool(
        np.array_equal(
            reconstructed_from_inverse,
            dev_g1,
        )
    )

    inverse_reconstruction_sorted = bool(
        np.array_equal(
            reconstructed_from_inverse,
            dev_g1[
                sort_idx
            ],
        )
    )


# ======================================================================================
# 8) Surviving working-weight checks
# ======================================================================================

working_weight_checks = {
    "working_weights_equal_derived_raw":
        bool(
            np.array_equal(
                working_weights,
                derived_raw_weights,
            )
        ),

    "working_weights_equal_derived_sorted":
        bool(
            np.array_equal(
                working_weights,
                derived_raw_weights[
                    sort_idx
                ],
            )
        ),

    "historical_reference_weights_equal_working":
        bool(
            np.array_equal(
                historical_weights,
                working_weights,
            )
        ),

    "historical_reference_weights_equal_working_sorted":
        bool(
            np.array_equal(
                historical_weights,
                working_weights[
                    sort_idx
                ],
            )
        ),
}


# ======================================================================================
# 9) Recover exact historical construction-helper source
# ======================================================================================

construction_helper = (
    apply_engine_balanced_g2_628fcr2
)

if not callable(
    construction_helper
):
    raise RuntimeError(
        "Historical engine-balanced G2 helper is not callable."
    )

try:

    construction_helper_signature = str(
        inspect.signature(
            construction_helper
        )
    )

except (TypeError, ValueError):

    construction_helper_signature = "<unavailable>"

try:

    construction_helper_source = inspect.getsource(
        construction_helper
    )

    construction_helper_source_available = True

    construction_helper_source_sha256 = hashlib.sha256(
        construction_helper_source.encode(
            "utf-8"
        )
    ).hexdigest()

except (OSError, TypeError):

    construction_helper_source = None
    construction_helper_source_available = False
    construction_helper_source_sha256 = None


# ======================================================================================
# 10) Conservative resolution
# ======================================================================================

row_exact = bool(
    (
        FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C[
            "values_exact"
        ]
        &
        FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C[
            "weights_exact"
        ]
    ).any()
)

row_close = bool(
    (
        FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C[
            "values_close_1e-15"
        ]
        &
        FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C[
            "weights_close_1e-15"
        ]
    ).any()
)

grouped_distribution_exact = bool(
    group_unique_values_exact
    and
    group_weights_exact
)

grouped_distribution_close = bool(
    group_unique_values_close
    and
    group_weights_close
)

FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C = bool(
    grouped_distribution_exact
    or
    grouped_distribution_close
)

FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C = bool(
    row_exact
    or
    row_close
)


# ======================================================================================
# 11) Audit fingerprint
# ======================================================================================

audit_payload = json.dumps(
    {
        "sort_idx_exact_mergesort":
            sort_idx_exact_mergesort,

        "inverse_range_valid":
            inverse_range_valid,

        "inverse_reconstructs_raw_dev_g1":
            inverse_reconstruction_raw,

        "inverse_reconstructs_sorted_dev_g1":
            inverse_reconstruction_sorted,

        "working_weight_checks":
            working_weight_checks,

        "row_pairing_rows":
            pairing_rows,

        "group_unique_values_exact":
            group_unique_values_exact,

        "group_weights_exact":
            group_weights_exact,

        "group_unique_values_close":
            group_unique_values_close,

        "group_weights_close":
            group_weights_close,

        "stored_unique_values_match_hist":
            stored_unique_values_match_hist,

        "stored_unique_weights_match_hist":
            stored_unique_weights_match_hist,

        "stored_unique_values_match_candidate":
            stored_unique_values_match_candidate,

        "stored_unique_weights_match_candidate":
            stored_unique_weights_match_candidate,

        "construction_helper_signature":
            construction_helper_signature,

        "construction_helper_source_sha256":
            construction_helper_source_sha256,

        "row_pairing_resolved":
            FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C,

        "weighted_distribution_resolved":
            FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C,

        "fd003_g2_created":
            False,

        "final_selective_risk_created":
            False,

        "dev_labels_used":
            False,

        "dev_error_scoring_performed":
            False,

        "model_inference_performed":
            False,

        "mc_execution_performed":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    },
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C = (
    hashlib.sha256(
        audit_payload
    ).hexdigest()
)


FD003_G2_REFERENCE_CREATED_629TR1C = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629TR1C = False
FD003_DEV_LABELS_USED_629TR1C = False
FD003_DEV_SCORING_PERFORMED_629TR1C = False
FD003_MODEL_INFERENCE_PERFORMED_629TR1C = False
FD003_MC_EXECUTION_PERFORMED_629TR1C = False
FD003_THRESHOLD_SELECTED_629TR1C = False
FD003_TEST_CONTENTS_READ_629TR1C = False
FD003_RUL_CONTENTS_READ_629TR1C = False

all_checks_passed_629tr1c = True


# ======================================================================================
# 12) Output
# ======================================================================================

print("=" * 136)

print(
    "STEP 6.29T-R1-CORRECTED — FD003 HISTORICAL G2 "
    "WEIGHTED-REFERENCE PAIRING / GROUPING RECOVERY"
)

print("=" * 136)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- CORRECTION TO FAILED R1 ASSUMPTION ---")

print(
    "sort_idx is a permutation                 :",
    True,
)

print(
    "inverse treated as unique-group index     :",
    True,
)

print(
    "inverse range valid                       :",
    inverse_range_valid,
)

print(
    "inverse reconstructs raw DEV G1           :",
    inverse_reconstruction_raw,
)

print(
    "inverse reconstructs sort_idx-ordered G1  :",
    inverse_reconstruction_sorted,
)

print(
    "sort_idx exact stable argsort(dev_g1)     :",
    sort_idx_exact_mergesort,
)

print("\n--- WORKING WEIGHT CHECKS ---")

for key, value in working_weight_checks.items():

    print(
        f"{key:48s}:",
        value,
    )

print("\n--- ROW-ORDER PAIRING PATHWAYS ---")

print(
    FD003_HISTORICAL_G2_ROW_PAIRING_AUDIT_629TR1C.to_string(
        index=False
    )
)

print("\n--- WEIGHTED EMPIRICAL DISTRIBUTION ---")

print(
    "Historical unique reference values       :",
    len(
        hist_unique_values
    ),
)

print(
    "Candidate unique reference values        :",
    len(
        candidate_unique_values
    ),
)

print(
    "Unique values exact                      :",
    group_unique_values_exact,
)

print(
    "Grouped weight masses exact              :",
    group_weights_exact,
)

print(
    "Unique values close @1e-15               :",
    group_unique_values_close,
)

print(
    "Grouped weight masses close @1e-15       :",
    group_weights_close,
)

print(
    "Stored unique values match historical    :",
    stored_unique_values_match_hist,
)

print(
    "Stored unique weights match historical   :",
    stored_unique_weights_match_hist,
)

print(
    "Stored unique values match candidate     :",
    stored_unique_values_match_candidate,
)

print(
    "Stored unique weights match candidate    :",
    stored_unique_weights_match_candidate,
)

print("\n--- HISTORICAL CONSTRUCTION HELPER ---")

print(
    "Signature                      :",
    construction_helper_signature,
)

print(
    "Source available               :",
    construction_helper_source_available,
)

print(
    "Source SHA-256                 :",
    construction_helper_source_sha256,
)

if construction_helper_source_available:

    print("\nSOURCE:\n")

    print(
        construction_helper_source
    )

print("\n--- RESOLUTION ---")

print(
    "Rowwise value-weight pairing recovered       :",
    FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C,
)

print(
    "Weighted empirical distribution recovered   :",
    FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C,
)

print(
    "Audit SHA-256                              :",
    FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C,
)

print(
    "\nPASS — the failed R1 assumption has been corrected. "
    "inverse_628fcr2 is audited as a unique-value group index, "
    "not as a permutation."
)

print(
    "No FD003 G2 reference, final selective risk, threshold, "
    "model inference, MC execution, TEST, or official RUL was used."
)

In [ ]:
# STEP 6.29U
# FD003 clean-DEV engine-membership + equal-engine-mass G2 input contract lock
#
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO G2 transform yet / NO threshold selection.

import hashlib
import json
import numpy as np
import pandas as pd


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629s", False)):
    raise RuntimeError(
        "STEP 6.29S must pass before STEP 6.29U."
    )

if not bool(globals().get("all_checks_passed_629tr1c", False)):
    raise RuntimeError(
        "STEP 6.29T-R1-CORRECTED must pass before STEP 6.29U."
    )

required_629u = [
    "FD003_G1_EVIDENCE_CONTRACT_SHA256_629S",
    "FD003_CLEAN_DEV_G1_RISK_629S",
    "FD003_CLEAN_DEV_G1_RISK_SHA256_629S",
    "FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C",
    "FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C",
    "FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C",
]

missing_629u = [
    name
    for name in required_629u
    if name not in globals()
]

if missing_629u:
    raise RuntimeError(
        "Missing required G2 provenance object(s): "
        + ", ".join(missing_629u)
    )

if not bool(
    FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C
):
    raise RuntimeError(
        "Historical G2 weighted empirical distribution is not resolved."
    )

if not bool(
    FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C
):
    raise RuntimeError(
        "Historical G2 rowwise pairing is not resolved."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629TR1C", True)):
    raise RuntimeError(
        "Unexpected TEST-access state after STEP 6.29T-R1-CORRECTED."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629TR1C", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state after STEP 6.29T-R1-CORRECTED."
    )

if bool(globals().get("FD003_G2_INPUT_CONTRACT_LOCKED_629U", False)):
    raise RuntimeError(
        "STEP 6.29U has already locked the FD003 G2 input contract."
    )


# ======================================================================================
# 2) Frozen FD003 DEV population contract from STEP 6.29C / 6.29F
# ======================================================================================

EXPECTED_DEV_ENGINES_629U = np.asarray(
    [
        3, 5, 7, 8, 15,
        31, 39, 41, 44, 48,
        55, 63, 65, 66, 70,
        78, 81, 96, 98, 99,
    ],
    dtype=np.int64,
)

EXPECTED_DEV_WINDOWS_629U = 4706
EXPECTED_DEV_ENGINE_COUNT_629U = 20


# ======================================================================================
# 3) Helpers
# ======================================================================================

def _raw_sha256_629u(value):

    arr_629u = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629u.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29U hashing."
        )

    return hashlib.sha256(
        arr_629u.tobytes(order="C")
    ).hexdigest()


def _normalize_engine_ids_629u(values):

    raw_629u = np.asarray(
        values
    ).reshape(-1)

    if len(raw_629u) != EXPECTED_DEV_WINDOWS_629U:
        return None

    try:
        as_float_629u = raw_629u.astype(
            np.float64
        )
    except Exception:
        return None

    if not np.isfinite(
        as_float_629u
    ).all():
        return None

    rounded_629u = np.rint(
        as_float_629u
    )

    if not np.array_equal(
        as_float_629u,
        rounded_629u
    ):
        return None

    ids_629u = rounded_629u.astype(
        np.int64
    )

    if not np.array_equal(
        np.sort(
            np.unique(
                ids_629u
            )
        ),
        EXPECTED_DEV_ENGINES_629U,
    ):
        return None

    return ids_629u


# ======================================================================================
# 4) Strict, name-filtered DEV key/metadata inventory
# ======================================================================================

forbidden_tokens_629u = (
    "test",
    "rul",
    "label",
    "target",
    "error",
    "mae",
    "score",
    "scoring",
    "pred",
    "prediction",
    "mc",
    "risk",
    "g1",
    "g2",
)

candidate_names_629u = []

for name_629u in sorted(
    globals().keys()
):

    lower_629u = name_629u.lower()

    fd003_related_629u = (
        "fd003" in lower_629u
        or
        "629f" in lower_629u
        or
        "629l" in lower_629u
    )

    dev_related_629u = (
        "dev" in lower_629u
    )

    key_related_629u = any(
        token_629u in lower_629u
        for token_629u in (
            "key",
            "window",
            "meta",
            "endpoint",
        )
    )

    forbidden_629u = any(
        token_629u in lower_629u
        for token_629u in forbidden_tokens_629u
    )

    if (
        fd003_related_629u
        and dev_related_629u
        and key_related_629u
        and not forbidden_629u
    ):
        candidate_names_629u.append(
            name_629u
        )


inventory_rows_629u = []
valid_engine_sources_629u = {}

engine_column_aliases_629u = (
    "unit_id",
    "engine_id",
    "unit",
    "engine",
)

for name_629u in candidate_names_629u:

    obj_629u = globals()[
        name_629u
    ]

    row_629u = {
        "name":
            name_629u,

        "type":
            type(
                obj_629u
            ).__name__,

        "shape":
            None,

        "columns_or_fields":
            None,

        "engine_field":
            None,

        "valid_fd003_dev_membership":
            False,
    }

    if isinstance(
        obj_629u,
        pd.DataFrame,
    ):

        row_629u[
            "shape"
        ] = tuple(
            int(x)
            for x in obj_629u.shape
        )

        columns_629u = [
            str(c)
            for c in obj_629u.columns
        ]

        row_629u[
            "columns_or_fields"
        ] = columns_629u

        lower_to_actual_629u = {
            str(c).lower(): c
            for c in obj_629u.columns
        }

        selected_column_629u = None

        for alias_629u in engine_column_aliases_629u:
            if alias_629u in lower_to_actual_629u:
                selected_column_629u = lower_to_actual_629u[
                    alias_629u
                ]
                break

        if (
            selected_column_629u is not None
            and len(
                obj_629u
            ) == EXPECTED_DEV_WINDOWS_629U
        ):

            ids_629u = _normalize_engine_ids_629u(
                obj_629u[
                    selected_column_629u
                ].to_numpy()
            )

            if ids_629u is not None:

                source_key_629u = (
                    f"{name_629u}.{selected_column_629u}"
                )

                valid_engine_sources_629u[
                    source_key_629u
                ] = ids_629u

                row_629u[
                    "engine_field"
                ] = str(
                    selected_column_629u
                )

                row_629u[
                    "valid_fd003_dev_membership"
                ] = True

        inventory_rows_629u.append(
            row_629u
        )

        continue

    try:
        arr_629u = np.asarray(
            obj_629u
        )
    except Exception:

        inventory_rows_629u.append(
            row_629u
        )

        continue

    row_629u[
        "shape"
    ] = tuple(
        int(x)
        for x in arr_629u.shape
    )

    field_names_629u = (
        list(
            arr_629u.dtype.names
        )
        if arr_629u.dtype.names is not None
        else None
    )

    row_629u[
        "columns_or_fields"
    ] = field_names_629u

    if (
        field_names_629u is not None
        and len(
            arr_629u
        ) == EXPECTED_DEV_WINDOWS_629U
    ):

        lower_to_actual_field_629u = {
            str(field_629u).lower():
                field_629u
            for field_629u
            in field_names_629u
        }

        selected_field_629u = None

        for alias_629u in engine_column_aliases_629u:
            if alias_629u in lower_to_actual_field_629u:
                selected_field_629u = lower_to_actual_field_629u[
                    alias_629u
                ]
                break

        if selected_field_629u is not None:

            ids_629u = _normalize_engine_ids_629u(
                arr_629u[
                    selected_field_629u
                ]
            )

            if ids_629u is not None:

                source_key_629u = (
                    f"{name_629u}.{selected_field_629u}"
                )

                valid_engine_sources_629u[
                    source_key_629u
                ] = ids_629u

                row_629u[
                    "engine_field"
                ] = str(
                    selected_field_629u
                )

                row_629u[
                    "valid_fd003_dev_membership"
                ] = True

    inventory_rows_629u.append(
        row_629u
    )


FD003_DEV_KEY_INVENTORY_629U = pd.DataFrame(
    inventory_rows_629u
)


# ======================================================================================
# 5) Resolve one authoritative engine-membership vector
# ======================================================================================

if len(
    valid_engine_sources_629u
) == 0:
    raise RuntimeError(
        "No safe 4706-row FD003 DEV key/metadata object with the exact "
        "20-engine membership was recovered. Do not construct G2 yet."
    )

valid_source_names_629u = sorted(
    valid_engine_sources_629u.keys()
)

canonical_source_name_629u = (
    valid_source_names_629u[0]
)

canonical_engine_ids_629u = (
    valid_engine_sources_629u[
        canonical_source_name_629u
    ]
)

all_valid_sources_identical_629u = all(
    np.array_equal(
        canonical_engine_ids_629u,
        engine_ids_629u,
    )
    for engine_ids_629u
    in valid_engine_sources_629u.values()
)

if not all_valid_sources_identical_629u:
    raise RuntimeError(
        "Multiple valid FD003 DEV engine-membership objects disagree "
        "row-by-row. Authority must be resolved before constructing G2."
    )


# ======================================================================================
# 6) Equal-engine-mass weighting contract
# ======================================================================================

FD003_DEV_ENGINE_IDS_629U = np.asarray(
    canonical_engine_ids_629u,
    dtype=np.int64,
).copy()

unique_engines_629u, counts_629u = np.unique(
    FD003_DEV_ENGINE_IDS_629U,
    return_counts=True,
)

if not np.array_equal(
    unique_engines_629u,
    EXPECTED_DEV_ENGINES_629U,
):
    raise RuntimeError(
        "Resolved FD003 DEV engine IDs do not match the frozen split."
    )

if len(
    unique_engines_629u
) != EXPECTED_DEV_ENGINE_COUNT_629U:
    raise RuntimeError(
        "Resolved FD003 DEV engine count is not 20."
    )

FD003_DEV_ENGINE_COUNT_MAP_629U = {
    int(engine_629u):
        int(count_629u)
    for engine_629u, count_629u
    in zip(
        unique_engines_629u,
        counts_629u,
    )
}

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U = np.asarray(
    [
        1.0
        /
        (
            EXPECTED_DEV_ENGINE_COUNT_629U
            *
            FD003_DEV_ENGINE_COUNT_MAP_629U[
                int(
                    engine_629u
                )
            ]
        )
        for engine_629u
        in FD003_DEV_ENGINE_IDS_629U
    ],
    dtype=np.float64,
)

if not np.isclose(
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD003 equal-engine-mass G2 weights do not sum to 1."
    )

engine_mass_629u = np.asarray(
    [
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U[
            FD003_DEV_ENGINE_IDS_629U
            ==
            engine_629u
        ].sum()
        for engine_629u
        in unique_engines_629u
    ],
    dtype=np.float64,
)

equal_engine_mass_exact_629u = bool(
    np.allclose(
        engine_mass_629u,
        np.full(
            EXPECTED_DEV_ENGINE_COUNT_629U,
            1.0
            /
            EXPECTED_DEV_ENGINE_COUNT_629U,
            dtype=np.float64,
        ),
        rtol=0.0,
        atol=1e-15,
    )
)

if not equal_engine_mass_exact_629u:
    raise RuntimeError(
        "FD003 G2 weights do not give equal total mass to each DEV engine."
    )


# ======================================================================================
# 7) Cross-check clean DEV G1 population alignment
# ======================================================================================

clean_dev_g1_629u = np.asarray(
    FD003_CLEAN_DEV_G1_RISK_629S,
    dtype=np.float64,
).reshape(-1)

if clean_dev_g1_629u.shape != (
    EXPECTED_DEV_WINDOWS_629U,
):
    raise RuntimeError(
        "FD003 clean DEV G1 population does not align with DEV key rows."
    )

if (
    _raw_sha256_629u(
        clean_dev_g1_629u
    )
    !=
    FD003_CLEAN_DEV_G1_RISK_SHA256_629S
):
    raise RuntimeError(
        "FD003 clean DEV G1 fingerprint changed before G2 input lock."
    )


# ======================================================================================
# 8) Freeze + fingerprints
# ======================================================================================

FD003_DEV_ENGINE_IDS_629U.setflags(
    write=False
)

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U.setflags(
    write=False
)

FD003_DEV_ENGINE_IDS_SHA256_629U = _raw_sha256_629u(
    FD003_DEV_ENGINE_IDS_629U
)

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629U = (
    _raw_sha256_629u(
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U
    )
)


# ======================================================================================
# 9) G2 input contract
# ======================================================================================

FD003_G2_INPUT_CONTRACT_629U = {
    "step":
        "6.29U",

    "dataset":
        "FD003",

    "clean_dev_population":
        "all_4706_DEV_windows_as_candidate_truncated_endpoints",

    "clean_dev_g1_evidence_contract_sha256":
        FD003_G1_EVIDENCE_CONTRACT_SHA256_629S,

    "historical_g2_grouping_audit_sha256":
        FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C,

    "historical_row_pairing_resolved":
        True,

    "historical_weighted_distribution_resolved":
        True,

    "dev_engine_membership_source":
        canonical_source_name_629u,

    "all_valid_engine_sources_identical":
        all_valid_sources_identical_629u,

    "dev_windows":
        EXPECTED_DEV_WINDOWS_629U,

    "dev_engines":
        EXPECTED_DEV_ENGINE_COUNT_629U,

    "engine_ids":
        [
            int(x)
            for x in unique_engines_629u
        ],

    "engine_window_counts":
        {
            str(k):
                int(v)
            for k, v
            in FD003_DEV_ENGINE_COUNT_MAP_629U.items()
        },

    "weight_formula":
        "1 / (n_dev_engines * windows_for_that_engine)",

    "equal_total_mass_per_engine":
        True,

    "engine_ids_sha256":
        FD003_DEV_ENGINE_IDS_SHA256_629U,

    "raw_equal_engine_weights_sha256":
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629U,

    "g2_reference_created":
        False,

    "final_selective_risk_created":
        False,

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "threshold_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629u = json.dumps(
    FD003_G2_INPUT_CONTRACT_629U,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_G2_INPUT_CONTRACT_SHA256_629U = hashlib.sha256(
    contract_payload_629u
).hexdigest()

FD003_G2_INPUT_CONTRACT_LOCKED_629U = True

FD003_G2_REFERENCE_CREATED_629U = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629U = False
FD003_DEV_LABELS_USED_629U = False
FD003_DEV_SCORING_PERFORMED_629U = False
FD003_MODEL_INFERENCE_PERFORMED_629U = False
FD003_MC_EXECUTION_PERFORMED_629U = False
FD003_THRESHOLD_SELECTED_629U = False
FD003_TEST_CONTENTS_READ_629U = False
FD003_RUL_CONTENTS_READ_629U = False

all_checks_passed_629u = True


# ======================================================================================
# 10) Output
# ======================================================================================

print("=" * 136)

print(
    "STEP 6.29U — FD003 CLEAN-DEV ENGINE-MEMBERSHIP + "
    "EQUAL-ENGINE-MASS G2 INPUT CONTRACT LOCK"
)

print("=" * 136)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- STRICT DEV KEY/METADATA INVENTORY ---")

if len(
    FD003_DEV_KEY_INVENTORY_629U
) == 0:

    print(
        "<no name-filtered DEV key/metadata candidates>"
    )

else:

    print(
        FD003_DEV_KEY_INVENTORY_629U.to_string(
            index=False
        )
    )

print("\n--- AUTHORITATIVE DEV ENGINE MEMBERSHIP ---")

print(
    "Valid source count             :",
    len(
        valid_engine_sources_629u
    ),
)

print(
    "Chosen source                  :",
    canonical_source_name_629u,
)

print(
    "All valid sources identical    :",
    all_valid_sources_identical_629u,
)

print(
    "DEV windows                    :",
    len(
        FD003_DEV_ENGINE_IDS_629U
    ),
)

print(
    "DEV engines                    :",
    len(
        unique_engines_629u
    ),
)

print(
    "Engine IDs                     :",
    unique_engines_629u.tolist(),
)

print(
    "Window counts by engine        :",
    FD003_DEV_ENGINE_COUNT_MAP_629U,
)

print("\n--- EQUAL-ENGINE-MASS WEIGHT CONTRACT ---")

print(
    "Weight sum                     :",
    float(
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U.sum()
    ),
)

print(
    "Unique raw weights             :",
    int(
        len(
            np.unique(
                FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629U
            )
        )
    ),
)

print(
    "Equal total mass per engine    :",
    equal_engine_mass_exact_629u,
)

print(
    "Per-engine mass min/max        :",
    (
        float(
            engine_mass_629u.min()
        ),
        float(
            engine_mass_629u.max()
        ),
    ),
)

print("\n--- FINGERPRINTS ---")

print(
    "DEV engine IDs SHA-256         :",
    FD003_DEV_ENGINE_IDS_SHA256_629U,
)

print(
    "Raw G2 weights SHA-256         :",
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629U,
)

print(
    "G2 input contract SHA-256      :",
    FD003_G2_INPUT_CONTRACT_SHA256_629U,
)

print(
    "\nPASS — the FD003 clean-DEV engine-membership and exact "
    "equal-engine-mass G2 weighting inputs are now locked."
)

print(
    "No G2 transform, final selective risk, threshold selection, "
    "DEV label/error scoring, TEST, official RUL, inference, or MC execution occurred."
)

print(
    "Next: sort clean-DEV G1 values stably with these paired weights, "
    "aggregate exact tie masses, freeze the FD003 G2 weighted reference, "
    "and apply the recovered weighted-midrank transform to obtain the "
    "final clean selective risk. Threshold selection will still remain forbidden."
)

In [ ]:
# STEP 6.29U-R1
# FD003 SAFE DEV KEY / ENGINE-MEMBERSHIP OBJECT LOCATOR
#
# READ-ONLY NAMESPACE METADATA AUDIT ONLY.
# NO array contents printed.
# NO model inference / NO MC execution / NO labels / NO scoring.
# NO TEST / NO official RUL / NO G2 construction / NO threshold selection.

import numpy as np
import pandas as pd

print("=" * 132)
print("STEP 6.29U-R1 — FD003 SAFE DEV KEY / ENGINE-MEMBERSHIP OBJECT LOCATOR")
print("=" * 132)

print("Model inference performed      : False")
print("MC-dropout execution performed : False")
print("DEV labels used                : False")
print("DEV scoring performed          : False")
print("FD003 G2 reference created     : False")
print("Threshold selected             : False")
print("TEST contents read             : False")
print("Official RUL contents read     : False")

forbidden_629ur1 = (
    "test",
    "rul",
    "label",
    "target",
    "error",
    "mae",
    "score",
    "scoring",
    "pred",
    "prediction",
    "risk",
    "mc",
)

keyish_629ur1 = (
    "key",
    "window",
    "engine",
    "unit",
    "meta",
    "membership",
)

rows_629ur1 = []

for name_629ur1 in sorted(globals().keys()):

    if name_629ur1.startswith("_"):
        continue

    lower_629ur1 = name_629ur1.lower()

    if any(
        token_629ur1 in lower_629ur1
        for token_629ur1 in forbidden_629ur1
    ):
        continue

    relevant_name_629ur1 = (
        (
            "dev" in lower_629ur1
            and any(
                token_629ur1 in lower_629ur1
                for token_629ur1 in keyish_629ur1
            )
        )
        or
        (
            "dev" in lower_629ur1
            and (
                "629f" in lower_629ur1
                or
                "629c" in lower_629ur1
            )
        )
    )

    if not relevant_name_629ur1:
        continue

    obj_629ur1 = globals()[name_629ur1]

    row_629ur1 = {
        "name": name_629ur1,
        "type": type(obj_629ur1).__name__,
        "shape_or_len": None,
        "dtype": None,
        "columns_or_fields": None,
        "first_dim_4706": False,
    }

    if isinstance(obj_629ur1, pd.DataFrame):

        row_629ur1["shape_or_len"] = tuple(
            int(x)
            for x in obj_629ur1.shape
        )

        row_629ur1["columns_or_fields"] = [
            str(c)
            for c in obj_629ur1.columns
        ]

        row_629ur1["first_dim_4706"] = bool(
            len(obj_629ur1) == 4706
        )

    elif isinstance(obj_629ur1, np.ndarray):

        row_629ur1["shape_or_len"] = tuple(
            int(x)
            for x in obj_629ur1.shape
        )

        row_629ur1["dtype"] = str(
            obj_629ur1.dtype
        )

        if obj_629ur1.dtype.names is not None:
            row_629ur1["columns_or_fields"] = [
                str(x)
                for x in obj_629ur1.dtype.names
            ]

        row_629ur1["first_dim_4706"] = bool(
            obj_629ur1.ndim >= 1
            and
            obj_629ur1.shape[0] == 4706
        )

    elif isinstance(obj_629ur1, dict):

        row_629ur1["shape_or_len"] = int(
            len(obj_629ur1)
        )

        row_629ur1["columns_or_fields"] = [
            str(k)
            for k in list(obj_629ur1.keys())[:50]
        ]

    elif isinstance(obj_629ur1, (list, tuple)):

        row_629ur1["shape_or_len"] = int(
            len(obj_629ur1)
        )

        row_629ur1["first_dim_4706"] = bool(
            len(obj_629ur1) == 4706
        )

    else:

        row_629ur1["shape_or_len"] = "<not inspected>"

    rows_629ur1.append(
        row_629ur1
    )

FD003_DEV_KEY_OBJECT_LOCATOR_629UR1 = pd.DataFrame(
    rows_629ur1
)

print("\n--- SAFE CANDIDATE INVENTORY ---")

if len(FD003_DEV_KEY_OBJECT_LOCATOR_629UR1) == 0:
    print("<no candidate names found>")
else:
    print(
        FD003_DEV_KEY_OBJECT_LOCATOR_629UR1.to_string(
            index=False
        )
    )

print("\n--- 4706-ROW CANDIDATES ---")

if (
    len(FD003_DEV_KEY_OBJECT_LOCATOR_629UR1) > 0
    and
    bool(
        FD003_DEV_KEY_OBJECT_LOCATOR_629UR1[
            "first_dim_4706"
        ].any()
    )
):
    print(
        FD003_DEV_KEY_OBJECT_LOCATOR_629UR1[
            FD003_DEV_KEY_OBJECT_LOCATOR_629UR1[
                "first_dim_4706"
            ]
        ].to_string(
            index=False
        )
    )
else:
    print("<none identified from metadata>")

print(
    "\nPASS — locator audit only. No G2 object was constructed and no frozen state was modified."
)
print(
    "Paste this output back before any further G2 step."
)

In [ ]:
# STEP 6.29U-CORRECTED
# FD003 strict-whitelist clean-DEV engine-membership + equal-engine-mass G2 input contract lock
#
# Uses the exact FD003 DEV window-unit vector recovered by STEP 6.29U-R1.
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO G2 transform / NO threshold selection.

import hashlib
import json
import numpy as np


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629s", False)):
    raise RuntimeError(
        "STEP 6.29S must pass before STEP 6.29U-CORRECTED."
    )

if not bool(globals().get("all_checks_passed_629tr1c", False)):
    raise RuntimeError(
        "STEP 6.29T-R1-CORRECTED must pass before STEP 6.29U-CORRECTED."
    )

required_629uc = [
    "DEV_WINDOW_UNIT_IDS_FD003_629F",
    "FD003_DEV_ENGINE_IDS_629C",
    "dev_engine_membership_exact_629f",
    "expected_dev_windows_629f",
    "FD003_CLEAN_DEV_G1_RISK_629S",
    "FD003_CLEAN_DEV_G1_RISK_SHA256_629S",
    "FD003_G1_EVIDENCE_CONTRACT_SHA256_629S",
    "FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C",
    "FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C",
    "FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C",
]

missing_629uc = [
    name
    for name in required_629uc
    if name not in globals()
]

if missing_629uc:
    raise RuntimeError(
        "Missing required STEP 6.29U-CORRECTED object(s): "
        + ", ".join(missing_629uc)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629TR1C", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29U-CORRECTED."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629TR1C", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29U-CORRECTED."
    )

if bool(globals().get("FD003_G2_INPUT_CONTRACT_LOCKED_629UC", False)):
    raise RuntimeError(
        "STEP 6.29U-CORRECTED has already been completed."
    )

if not bool(
    FD003_HISTORICAL_G2_WEIGHTED_DISTRIBUTION_RESOLVED_629TR1C
):
    raise RuntimeError(
        "Historical G2 weighted-distribution semantics are not resolved."
    )

if not bool(
    FD003_HISTORICAL_G2_ROW_PAIRING_RESOLVED_629TR1C
):
    raise RuntimeError(
        "Historical G2 row-pairing semantics are not resolved."
    )


# ======================================================================================
# 2) Helpers
# ======================================================================================

def _raw_sha256_629uc(value):

    arr_629uc = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629uc.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29U-CORRECTED hashing."
        )

    return hashlib.sha256(
        arr_629uc.tobytes(order="C")
    ).hexdigest()


# ======================================================================================
# 3) Strict-whitelist authoritative FD003 DEV engine membership
# ======================================================================================

dev_window_unit_ids_629uc = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD003_629F,
    dtype=np.int64,
).reshape(-1)

frozen_dev_engine_ids_629uc = np.asarray(
    FD003_DEV_ENGINE_IDS_629C,
    dtype=np.int64,
).reshape(-1)

expected_dev_windows_629uc = int(
    expected_dev_windows_629f
)

if expected_dev_windows_629uc != 4706:
    raise RuntimeError(
        "Frozen FD003 DEV expected-window count is not 4706."
    )

if dev_window_unit_ids_629uc.shape != (
    expected_dev_windows_629uc,
):
    raise RuntimeError(
        "DEV_WINDOW_UNIT_IDS_FD003_629F does not contain exactly 4706 window memberships."
    )

if frozen_dev_engine_ids_629uc.shape != (20,):
    raise RuntimeError(
        "Frozen FD003 DEV engine-ID vector is not length 20."
    )

if len(
    np.unique(
        frozen_dev_engine_ids_629uc
    )
) != 20:
    raise RuntimeError(
        "Frozen FD003 DEV engine-ID vector is not unique."
    )

if not bool(
    dev_engine_membership_exact_629f
):
    raise RuntimeError(
        "STEP 6.29F did not certify exact DEV engine membership."
    )

window_unique_engines_629uc = np.sort(
    np.unique(
        dev_window_unit_ids_629uc
    )
)

frozen_unique_engines_629uc = np.sort(
    frozen_dev_engine_ids_629uc
)

if not np.array_equal(
    window_unique_engines_629uc,
    frozen_unique_engines_629uc,
):
    raise RuntimeError(
        "FD003 DEV window engine memberships disagree with the frozen STEP 6.29C DEV split."
    )


# ======================================================================================
# 4) Verify row alignment with the already-locked clean DEV G1 population
# ======================================================================================

clean_dev_g1_629uc = np.asarray(
    FD003_CLEAN_DEV_G1_RISK_629S,
    dtype=np.float64,
).reshape(-1)

if clean_dev_g1_629uc.shape != (
    expected_dev_windows_629uc,
):
    raise RuntimeError(
        "FD003 clean DEV G1 population length does not match the FD003 DEV window-key population."
    )

if not np.isfinite(
    clean_dev_g1_629uc
).all():
    raise RuntimeError(
        "FD003 clean DEV G1 risk contains non-finite values."
    )

if (
    _raw_sha256_629uc(
        clean_dev_g1_629uc
    )
    !=
    FD003_CLEAN_DEV_G1_RISK_SHA256_629S
):
    raise RuntimeError(
        "FD003 clean DEV G1 fingerprint changed before the G2 input lock."
    )


# ======================================================================================
# 5) Exact equal-engine-mass raw weights
# ======================================================================================

unique_engines_629uc, counts_629uc = np.unique(
    dev_window_unit_ids_629uc,
    return_counts=True,
)

n_dev_engines_629uc = int(
    len(
        unique_engines_629uc
    )
)

if n_dev_engines_629uc != 20:
    raise RuntimeError(
        "Resolved FD003 DEV window population does not contain exactly 20 engines."
    )

engine_count_map_629uc = {
    int(engine_629uc): int(count_629uc)
    for engine_629uc, count_629uc
    in zip(
        unique_engines_629uc,
        counts_629uc,
    )
}

raw_equal_engine_weights_629uc = np.asarray(
    [
        1.0
        /
        (
            n_dev_engines_629uc
            *
            engine_count_map_629uc[
                int(engine_629uc)
            ]
        )
        for engine_629uc
        in dev_window_unit_ids_629uc
    ],
    dtype=np.float64,
)

if not np.isclose(
    raw_equal_engine_weights_629uc.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD003 equal-engine-mass weights do not sum to 1."
    )

per_engine_mass_629uc = np.asarray(
    [
        raw_equal_engine_weights_629uc[
            dev_window_unit_ids_629uc
            ==
            engine_629uc
        ].sum()
        for engine_629uc
        in unique_engines_629uc
    ],
    dtype=np.float64,
)

equal_engine_mass_629uc = bool(
    np.allclose(
        per_engine_mass_629uc,
        np.full(
            n_dev_engines_629uc,
            1.0 / n_dev_engines_629uc,
            dtype=np.float64,
        ),
        rtol=0.0,
        atol=1e-15,
    )
)

if not equal_engine_mass_629uc:
    raise RuntimeError(
        "FD003 G2 raw weights do not assign equal total mass to every DEV engine."
    )


# ======================================================================================
# 6) Freeze the G2-input artifacts
# ======================================================================================

FD003_DEV_WINDOW_UNIT_IDS_629UC = np.asarray(
    dev_window_unit_ids_629uc,
    dtype=np.int64,
).copy()

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC = np.asarray(
    raw_equal_engine_weights_629uc,
    dtype=np.float64,
).copy()

FD003_DEV_WINDOW_UNIT_IDS_629UC.setflags(
    write=False
)

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC.setflags(
    write=False
)

FD003_DEV_WINDOW_UNIT_IDS_SHA256_629UC = _raw_sha256_629uc(
    FD003_DEV_WINDOW_UNIT_IDS_629UC
)

FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629UC = _raw_sha256_629uc(
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC
)

FD003_DEV_ENGINE_COUNT_MAP_629UC = dict(
    engine_count_map_629uc
)


# ======================================================================================
# 7) G2 input contract
# ======================================================================================

FD003_G2_INPUT_CONTRACT_629UC = {
    "step":
        "6.29U-CORRECTED",

    "dataset":
        "FD003",

    "dev_window_engine_membership_source":
        "DEV_WINDOW_UNIT_IDS_FD003_629F",

    "dev_split_engine_ids_source":
        "FD003_DEV_ENGINE_IDS_629C",

    "dev_membership_certified_exact_in_629f":
        True,

    "dev_windows":
        int(
            expected_dev_windows_629uc
        ),

    "dev_engines":
        int(
            n_dev_engines_629uc
        ),

    "engine_ids":
        [
            int(x)
            for x in unique_engines_629uc
        ],

    "engine_window_counts":
        {
            str(k): int(v)
            for k, v
            in engine_count_map_629uc.items()
        },

    "weight_formula":
        "1 / (n_dev_engines * windows_for_that_engine)",

    "equal_total_mass_per_engine":
        True,

    "clean_dev_g1_evidence_contract_sha256":
        FD003_G1_EVIDENCE_CONTRACT_SHA256_629S,

    "historical_g2_grouping_audit_sha256":
        FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C,

    "historical_row_pairing_resolved":
        True,

    "historical_weighted_distribution_resolved":
        True,

    "dev_window_unit_ids_sha256":
        FD003_DEV_WINDOW_UNIT_IDS_SHA256_629UC,

    "raw_equal_engine_weights_sha256":
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629UC,

    "g2_reference_created":
        False,

    "final_selective_risk_created":
        False,

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "threshold_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629uc = json.dumps(
    FD003_G2_INPUT_CONTRACT_629UC,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_G2_INPUT_CONTRACT_SHA256_629UC = hashlib.sha256(
    contract_payload_629uc
).hexdigest()

FD003_G2_INPUT_CONTRACT_LOCKED_629UC = True

FD003_G2_REFERENCE_CREATED_629UC = False
FD003_FINAL_SELECTIVE_RISK_CREATED_629UC = False
FD003_DEV_LABELS_USED_629UC = False
FD003_DEV_SCORING_PERFORMED_629UC = False
FD003_MODEL_INFERENCE_PERFORMED_629UC = False
FD003_MC_EXECUTION_PERFORMED_629UC = False
FD003_THRESHOLD_SELECTED_629UC = False
FD003_TEST_CONTENTS_READ_629UC = False
FD003_RUL_CONTENTS_READ_629UC = False

all_checks_passed_629uc = True


# ======================================================================================
# 8) Output
# ======================================================================================

print("=" * 136)

print(
    "STEP 6.29U-CORRECTED — FD003 STRICT-WHITELIST CLEAN-DEV "
    "ENGINE-MEMBERSHIP + EQUAL-ENGINE-MASS G2 INPUT LOCK"
)

print("=" * 136)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", False)
print("Final selective risk created   :", False)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- AUTHORITATIVE DEV ENGINE MEMBERSHIP ---")

print(
    "Membership source              : DEV_WINDOW_UNIT_IDS_FD003_629F"
)

print(
    "DEV windows                    :",
    len(
        FD003_DEV_WINDOW_UNIT_IDS_629UC
    ),
)

print(
    "DEV engines                    :",
    n_dev_engines_629uc,
)

print(
    "Engine IDs                     :",
    unique_engines_629uc.tolist(),
)

print(
    "Matches frozen STEP 6.29C split:",
    np.array_equal(
        window_unique_engines_629uc,
        frozen_unique_engines_629uc,
    ),
)

print(
    "STEP 6.29F membership exact    :",
    bool(
        dev_engine_membership_exact_629f
    ),
)

print(
    "Window counts by engine        :",
    FD003_DEV_ENGINE_COUNT_MAP_629UC,
)

print("\n--- EQUAL-ENGINE-MASS WEIGHT CONTRACT ---")

print(
    "Weight formula                 : 1 / (20 * windows_for_that_engine)"
)

print(
    "Weight sum                     :",
    float(
        FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC.sum()
    ),
)

print(
    "Unique raw weights             :",
    int(
        len(
            np.unique(
                FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC
            )
        )
    ),
)

print(
    "Equal total mass per engine    :",
    equal_engine_mass_629uc,
)

print(
    "Per-engine mass min/max        :",
    (
        float(
            per_engine_mass_629uc.min()
        ),
        float(
            per_engine_mass_629uc.max()
        ),
    ),
)

print("\n--- FINGERPRINTS ---")

print(
    "DEV window-unit IDs SHA-256    :",
    FD003_DEV_WINDOW_UNIT_IDS_SHA256_629UC,
)

print(
    "Raw G2 weights SHA-256         :",
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629UC,
)

print(
    "G2 input contract SHA-256      :",
    FD003_G2_INPUT_CONTRACT_SHA256_629UC,
)

print(
    "\nPASS — the authoritative FD003 clean-DEV window engine membership "
    "and exact equal-engine-mass G2 weighting inputs are now frozen."
)

print(
    "No G2 transform, final selective risk, threshold selection, "
    "DEV label/error scoring, TEST, official RUL, model inference, "
    "or MC execution occurred."
)

print(
    "Next: construct the frozen FD003 G2 weighted empirical reference "
    "from the locked clean-DEV G1 values and these paired weights, "
    "then apply the recovered weighted-midrank transform. "
    "Threshold selection will still remain forbidden."
)

In [ ]:
# STEP 6.29V
# FD003 clean-DEV G2 weighted-reference construction + final selective-risk lock
#
# Uses only already-frozen clean-DEV G1 values and frozen equal-engine-mass weights.
# NO model inference / NO MC execution / NO DEV labels or error scoring.
# NO TEST / NO official RUL / NO threshold selection / NO retuning.

import hashlib
import json
import numpy as np


# ======================================================================================
# 1) Preconditions
# ======================================================================================

if not bool(globals().get("all_checks_passed_629s", False)):
    raise RuntimeError(
        "STEP 6.29S must pass before STEP 6.29V."
    )

if not bool(globals().get("all_checks_passed_629uc", False)):
    raise RuntimeError(
        "STEP 6.29U-CORRECTED must pass before STEP 6.29V."
    )

required_629v = [
    "FD003_CLEAN_DEV_G1_RISK_629S",
    "FD003_CLEAN_DEV_G1_RISK_SHA256_629S",
    "FD003_G1_EVIDENCE_CONTRACT_SHA256_629S",
    "FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC",
    "FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629UC",
    "FD003_G2_INPUT_CONTRACT_SHA256_629UC",
    "FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C",
]

missing_629v = [
    name
    for name in required_629v
    if name not in globals()
]

if missing_629v:
    raise RuntimeError(
        "Missing required STEP 6.29V object(s): "
        + ", ".join(missing_629v)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629UC", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29V."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629UC", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29V."
    )

if bool(globals().get("FD003_G2_REFERENCE_LOCKED_629V", False)):
    raise RuntimeError(
        "STEP 6.29V has already been completed."
    )


# ======================================================================================
# 2) Hash helper
# ======================================================================================

def _raw_sha256_629v(value):

    arr_629v = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629v.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29V hashing."
        )

    return hashlib.sha256(
        arr_629v.tobytes(order="C")
    ).hexdigest()


# ======================================================================================
# 3) Verify frozen G1 values + equal-engine weights unchanged
# ======================================================================================

g1_values_629v = np.asarray(
    FD003_CLEAN_DEV_G1_RISK_629S,
    dtype=np.float64,
).reshape(-1)

raw_weights_629v = np.asarray(
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_629UC,
    dtype=np.float64,
).reshape(-1)

if g1_values_629v.shape != (4706,):
    raise RuntimeError(
        "FD003 clean DEV G1 risk shape changed."
    )

if raw_weights_629v.shape != (4706,):
    raise RuntimeError(
        "FD003 raw equal-engine G2 weight shape changed."
    )

if not np.isfinite(g1_values_629v).all():
    raise RuntimeError(
        "FD003 clean DEV G1 risk contains non-finite values."
    )

if not np.isfinite(raw_weights_629v).all():
    raise RuntimeError(
        "FD003 G2 raw weights contain non-finite values."
    )

if bool((raw_weights_629v < 0.0).any()):
    raise RuntimeError(
        "FD003 G2 raw weights contain negative values."
    )

if not np.isclose(
    raw_weights_629v.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD003 G2 raw weights no longer sum to 1."
    )

if (
    _raw_sha256_629v(g1_values_629v)
    !=
    FD003_CLEAN_DEV_G1_RISK_SHA256_629S
):
    raise RuntimeError(
        "FD003 clean DEV G1 fingerprint changed."
    )

if (
    _raw_sha256_629v(raw_weights_629v)
    !=
    FD003_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_629UC
):
    raise RuntimeError(
        "FD003 G2 raw weight fingerprint changed."
    )


# ======================================================================================
# 4) Construct exact historical-semantics weighted reference
#
# Recovered rule:
#   1. stable-sort G1 values with kind='mergesort'
#   2. carry raw equal-engine weights through the same permutation
#   3. aggregate total weight mass for exact tied values
#   4. weighted midrank for an exact tie:
#          cumulative_mass_before_tie + 0.5 * tie_mass
# ======================================================================================

FD003_G2_SORT_INDEX_629V = np.argsort(
    g1_values_629v,
    kind="mergesort",
).astype(
    np.int64
)

FD003_G2_REFERENCE_VALUES_629V = np.asarray(
    g1_values_629v[
        FD003_G2_SORT_INDEX_629V
    ],
    dtype=np.float64,
)

FD003_G2_REFERENCE_WEIGHTS_629V = np.asarray(
    raw_weights_629v[
        FD003_G2_SORT_INDEX_629V
    ],
    dtype=np.float64,
)

if not bool(
    np.all(
        np.diff(
            FD003_G2_REFERENCE_VALUES_629V
        )
        >=
        0.0
    )
):
    raise RuntimeError(
        "FD003 G2 reference values are not sorted."
    )

if not np.isclose(
    FD003_G2_REFERENCE_WEIGHTS_629V.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Sorted FD003 G2 reference weights do not sum to 1."
    )

(
    FD003_G2_UNIQUE_VALUES_629V,
    FD003_G2_UNIQUE_INVERSE_629V,
) = np.unique(
    FD003_G2_REFERENCE_VALUES_629V,
    return_inverse=True,
)

FD003_G2_UNIQUE_VALUES_629V = np.asarray(
    FD003_G2_UNIQUE_VALUES_629V,
    dtype=np.float64,
)

FD003_G2_UNIQUE_INVERSE_629V = np.asarray(
    FD003_G2_UNIQUE_INVERSE_629V,
    dtype=np.int64,
)

FD003_G2_UNIQUE_WEIGHTS_629V = np.bincount(
    FD003_G2_UNIQUE_INVERSE_629V,
    weights=FD003_G2_REFERENCE_WEIGHTS_629V,
    minlength=len(
        FD003_G2_UNIQUE_VALUES_629V
    ),
).astype(
    np.float64
)

if not np.isclose(
    FD003_G2_UNIQUE_WEIGHTS_629V.sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD003 grouped G2 weight masses do not sum to 1."
    )

FD003_G2_CUMULATIVE_WEIGHTS_629V = np.cumsum(
    FD003_G2_UNIQUE_WEIGHTS_629V,
    dtype=np.float64,
)

FD003_G2_MID_MASS_629V = (
    FD003_G2_CUMULATIVE_WEIGHTS_629V
    -
    0.5
    *
    FD003_G2_UNIQUE_WEIGHTS_629V
)

if bool(
    (
        FD003_G2_MID_MASS_629V
        <
        0.0
    ).any()
    or
    (
        FD003_G2_MID_MASS_629V
        >
        1.0
    ).any()
):
    raise RuntimeError(
        "FD003 G2 mid-mass values lie outside [0,1]."
    )


# ======================================================================================
# 5) Recovered historical weighted-midrank transform
# ======================================================================================

def apply_fd003_g2_weighted_midrank_629v(values):

    x_629v = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    if not np.isfinite(
        x_629v
    ).all():
        raise ValueError(
            "FD003 G2 input contains non-finite values."
        )

    result_629v = np.empty(
        len(
            x_629v
        ),
        dtype=np.float64,
    )

    left_idx_629v = np.searchsorted(
        FD003_G2_UNIQUE_VALUES_629V,
        x_629v,
        side="left",
    )

    right_idx_629v = np.searchsorted(
        FD003_G2_UNIQUE_VALUES_629V,
        x_629v,
        side="right",
    )

    for i_629v in range(
        len(
            x_629v
        )
    ):

        li_629v = int(
            left_idx_629v[
                i_629v
            ]
        )

        ri_629v = int(
            right_idx_629v[
                i_629v
            ]
        )

        # Exact reference value / tie.
        if ri_629v > li_629v:

            result_629v[
                i_629v
            ] = FD003_G2_MID_MASS_629V[
                li_629v
            ]

        # Below reference minimum.
        elif li_629v == 0:

            result_629v[
                i_629v
            ] = 0.0

        # Above reference maximum.
        elif li_629v >= len(
            FD003_G2_UNIQUE_VALUES_629V
        ):

            result_629v[
                i_629v
            ] = 1.0

        # Between two observed reference values.
        else:

            result_629v[
                i_629v
            ] = FD003_G2_CUMULATIVE_WEIGHTS_629V[
                li_629v - 1
            ]

    return result_629v


# ======================================================================================
# 6) Apply G2 to clean DEV G1 => final clean selective risk
# ======================================================================================

FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V = (
    apply_fd003_g2_weighted_midrank_629v(
        g1_values_629v
    )
)

if (
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V.shape
    !=
    (4706,)
):
    raise RuntimeError(
        "FD003 final clean selective-risk shape mismatch."
    )

if not np.isfinite(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
).all():
    raise RuntimeError(
        "FD003 final clean selective risk contains non-finite values."
    )

if bool(
    (
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
        <
        0.0
    ).any()
    or
    (
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
        >
        1.0
    ).any()
):
    raise RuntimeError(
        "FD003 final clean selective risk lies outside [0,1]."
    )

reference_lookup_idx_629v = np.searchsorted(
    FD003_G2_UNIQUE_VALUES_629V,
    g1_values_629v,
    side="left",
)

expected_clean_risk_629v = FD003_G2_MID_MASS_629V[
    reference_lookup_idx_629v
]

if not np.array_equal(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
    expected_clean_risk_629v,
):
    raise RuntimeError(
        "FD003 clean DEV G2 transform does not reproduce exact reference mid-masses."
    )


# ======================================================================================
# 7) Freeze arrays + fingerprints
# ======================================================================================

for arr_629v in [
    FD003_G2_SORT_INDEX_629V,
    FD003_G2_REFERENCE_VALUES_629V,
    FD003_G2_REFERENCE_WEIGHTS_629V,
    FD003_G2_UNIQUE_VALUES_629V,
    FD003_G2_UNIQUE_INVERSE_629V,
    FD003_G2_UNIQUE_WEIGHTS_629V,
    FD003_G2_CUMULATIVE_WEIGHTS_629V,
    FD003_G2_MID_MASS_629V,
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
]:
    arr_629v.setflags(
        write=False
    )

FD003_G2_REFERENCE_VALUES_SHA256_629V = _raw_sha256_629v(
    FD003_G2_REFERENCE_VALUES_629V
)

FD003_G2_REFERENCE_WEIGHTS_SHA256_629V = _raw_sha256_629v(
    FD003_G2_REFERENCE_WEIGHTS_629V
)

FD003_G2_UNIQUE_VALUES_SHA256_629V = _raw_sha256_629v(
    FD003_G2_UNIQUE_VALUES_629V
)

FD003_G2_UNIQUE_WEIGHTS_SHA256_629V = _raw_sha256_629v(
    FD003_G2_UNIQUE_WEIGHTS_629V
)

FD003_G2_MID_MASS_SHA256_629V = _raw_sha256_629v(
    FD003_G2_MID_MASS_629V
)

FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V = _raw_sha256_629v(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
)


# ======================================================================================
# 8) Final clean-risk evidence contract
# ======================================================================================

FD003_CLEAN_SELECTIVE_RISK_CONTRACT_629V = {
    "step":
        "6.29V",

    "dataset":
        "FD003",

    "input":
        "frozen_clean_DEV_G1_risk",

    "g1_evidence_contract_sha256":
        FD003_G1_EVIDENCE_CONTRACT_SHA256_629S,

    "g2_input_contract_sha256":
        FD003_G2_INPUT_CONTRACT_SHA256_629UC,

    "historical_g2_grouping_audit_sha256":
        FD003_HISTORICAL_G2_GROUPING_AUDIT_SHA256_629TR1C,

    "reference_population":
        "all_4706_clean_DEV_windows_as_candidate_truncated_endpoints",

    "reference_weighting":
        "equal_total_mass_per_DEV_engine",

    "sorting":
        "stable_mergesort",

    "tie_handling":
        "aggregate_exact_value_weight_mass_then_weighted_midrank",

    "between_reference_values":
        "cumulative_weight_mass_below_query",

    "below_reference_minimum":
        0.0,

    "above_reference_maximum":
        1.0,

    "g2_reference_size":
        int(
            len(
                FD003_G2_REFERENCE_VALUES_629V
            )
        ),

    "g2_unique_value_count":
        int(
            len(
                FD003_G2_UNIQUE_VALUES_629V
            )
        ),

    "g2_reference_values_sha256":
        FD003_G2_REFERENCE_VALUES_SHA256_629V,

    "g2_reference_weights_sha256":
        FD003_G2_REFERENCE_WEIGHTS_SHA256_629V,

    "g2_unique_values_sha256":
        FD003_G2_UNIQUE_VALUES_SHA256_629V,

    "g2_unique_weights_sha256":
        FD003_G2_UNIQUE_WEIGHTS_SHA256_629V,

    "g2_mid_mass_sha256":
        FD003_G2_MID_MASS_SHA256_629V,

    "clean_dev_final_selective_risk_sha256":
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,

    "risk_direction":
        "larger_value_means_lower_reliability",

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "threshold_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

contract_payload_629v = json.dumps(
    FD003_CLEAN_SELECTIVE_RISK_CONTRACT_629V,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V = hashlib.sha256(
    contract_payload_629v
).hexdigest()


FD003_G2_REFERENCE_LOCKED_629V = True
FD003_G2_REFERENCE_CREATED_629V = True
FD003_FINAL_SELECTIVE_RISK_CREATED_629V = True

FD003_DEV_LABELS_USED_629V = False
FD003_DEV_SCORING_PERFORMED_629V = False
FD003_MODEL_INFERENCE_PERFORMED_629V = False
FD003_MC_EXECUTION_PERFORMED_629V = False
FD003_THRESHOLD_SELECTED_629V = False
FD003_TEST_CONTENTS_READ_629V = False
FD003_RUL_CONTENTS_READ_629V = False

all_checks_passed_629v = True


# ======================================================================================
# 9) Output
# ======================================================================================

print("=" * 136)

print(
    "STEP 6.29V — FD003 CLEAN-DEV G2 WEIGHTED-REFERENCE "
    "CONSTRUCTION + FINAL SELECTIVE-RISK LOCK"
)

print("=" * 136)

print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("FD003 G2 reference created     :", True)
print("Final selective risk created   :", True)
print("Threshold selected             :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- G2 WEIGHTED REFERENCE ---")

print(
    "Reference windows              :",
    len(
        FD003_G2_REFERENCE_VALUES_629V
    ),
)

print(
    "Unique reference values        :",
    len(
        FD003_G2_UNIQUE_VALUES_629V
    ),
)

print(
    "Reference values sorted        :",
    True,
)

print(
    "Reference weight sum           :",
    float(
        FD003_G2_REFERENCE_WEIGHTS_629V.sum()
    ),
)

print(
    "Grouped weight sum             :",
    float(
        FD003_G2_UNIQUE_WEIGHTS_629V.sum()
    ),
)

print(
    "Tie groups                     :",
    int(
        np.sum(
            np.bincount(
                FD003_G2_UNIQUE_INVERSE_629V
            )
            >
            1
        )
    ),
)

print("\n--- FINAL CLEAN SELECTIVE RISK ---")

print(
    "Shape                          :",
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V.shape,
)

print(
    "Min / median / mean / max      :",
    (
        float(
            FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V.min()
        ),
        float(
            np.median(
                FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
            )
        ),
        float(
            FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V.mean()
        ),
        float(
            FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V.max()
        ),
    ),
)

print(
    "Within [0,1]                   :",
    True,
)

print(
    "Exact reference-midmass check  :",
    True,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "G2 reference values SHA-256    :",
    FD003_G2_REFERENCE_VALUES_SHA256_629V,
)

print(
    "G2 reference weights SHA-256   :",
    FD003_G2_REFERENCE_WEIGHTS_SHA256_629V,
)

print(
    "G2 unique values SHA-256       :",
    FD003_G2_UNIQUE_VALUES_SHA256_629V,
)

print(
    "G2 unique weights SHA-256      :",
    FD003_G2_UNIQUE_WEIGHTS_SHA256_629V,
)

print(
    "G2 mid-mass SHA-256            :",
    FD003_G2_MID_MASS_SHA256_629V,
)

print(
    "Final clean risk SHA-256       :",
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,
)

print(
    "Clean-risk contract SHA-256    :",
    FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V,
)

print(
    "\nPASS — the FD003 clean-DEV G2 weighted empirical reference and "
    "final label-free selective-risk transform are now constructed, "
    "fingerprinted, and locked."
)

print(
    "No DEV labels/error scoring, threshold selection, model inference, "
    "MC execution, TEST, or official RUL was used in this step."
)

print(
    "Next: evaluate the frozen clean-DEV selective-risk ordering using "
    "DEV outcomes only as reporting/calibration evidence, then proceed "
    "to the controlled corruption robustness analysis before any final "
    "operating-threshold freeze."
)

In [ ]:
# STEP 6.29W
# FD003 CLEAN-DEV SELECTIVE-EVALUATION PROTOCOL FREEZE
#
# Freezes how the already-locked clean DEV reliability risk will be evaluated.
# NO DEV outcome/error values inspected in this step.
# NO threshold selection.
# NO model inference / MC execution.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629v", False)):
    raise RuntimeError(
        "STEP 6.29V must pass before STEP 6.29W."
    )

required_629w = [
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V",
]

missing_629w = [
    name
    for name in required_629w
    if name not in globals()
]

if missing_629w:
    raise RuntimeError(
        "Missing required STEP 6.29W object(s): "
        + ", ".join(missing_629w)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629V", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29W."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629V", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29W."
    )

if bool(
    globals().get(
        "FD003_CLEAN_DEV_EVAL_PROTOCOL_LOCKED_629W",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29W has already been completed."
    )


# =============================================================================
# 2) Hash helper
# =============================================================================

def _raw_sha256_629w(value):

    arr_629w = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629w.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29W hashing."
        )

    return hashlib.sha256(
        arr_629w.tobytes(order="C")
    ).hexdigest()


# =============================================================================
# 3) Verify frozen clean DEV risk unchanged
# =============================================================================

risk_629w = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
    dtype=np.float64,
).reshape(-1)

if risk_629w.shape != (4706,):
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk shape is not (4706,)."
    )

if not np.isfinite(risk_629w).all():
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk contains non-finite values."
    )

if not (
    (risk_629w >= 0.0)
    &
    (risk_629w <= 1.0)
).all():
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk lies outside [0,1]."
    )

if (
    _raw_sha256_629w(risk_629w)
    !=
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V
):
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk fingerprint changed "
        "before evaluation-protocol lock."
    )


# =============================================================================
# 4) Prospectively freeze clean-DEV evaluation convention
# =============================================================================

FD003_CLEAN_DEV_EVAL_COVERAGES_629W = np.asarray(
    [
        0.90,
        0.80,
        0.70,
    ],
    dtype=np.float64,
)

FD003_CLEAN_DEV_EVAL_COVERAGES_629W.setflags(
    write=False
)

FD003_CLEAN_DEV_EVAL_ACCEPT_COUNTS_629W = np.asarray(
    [
        int(
            np.ceil(
                float(coverage_629w)
                *
                len(risk_629w)
            )
        )
        for coverage_629w
        in FD003_CLEAN_DEV_EVAL_COVERAGES_629W
    ],
    dtype=np.int64,
)

FD003_CLEAN_DEV_EVAL_ACCEPT_COUNTS_629W.setflags(
    write=False
)


# =============================================================================
# 5) Freeze protocol contract
# =============================================================================

FD003_CLEAN_DEV_EVAL_PROTOCOL_629W = {

    "step":
        "6.29W",

    "dataset":
        "FD003",

    "population":
        "clean_DEV_windows",

    "population_size":
        int(len(risk_629w)),

    "risk_source":
        "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",

    "risk_contract_sha256":
        FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V,

    "risk_sha256":
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,

    "risk_orientation":
        "lower_risk_more_reliable_accept_first",

    "outcome_for_later_evaluation":
        "absolute_RUL_error_on_DEV_only",

    "primary_ordering_metric":
        "Spearman_correlation_risk_vs_absolute_error",

    "selective_gain_definition":
        "full_DEV_MAE_minus_accepted_subset_MAE",

    "diagnostic_coverages":
        [
            0.90,
            0.80,
            0.70,
        ],

    "coverage_selection_rule":
        "stable_ascending_risk_fixed_count",

    "fixed_count_rounding":
        "ceil(target_coverage * n_DEV_windows)",

    "tie_break_rule":
        "stable_original_DEV_window_order",

    "nominal_operating_coverage_reference":
        0.80,

    "threshold_selected":
        False,

    "dev_outcomes_inspected_in_this_step":
        False,

    "dev_error_scoring_performed_in_this_step":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "post_test_retuning_allowed":
        False,
}

payload_629w = json.dumps(
    FD003_CLEAN_DEV_EVAL_PROTOCOL_629W,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_CLEAN_DEV_EVAL_PROTOCOL_SHA256_629W = hashlib.sha256(
    payload_629w
).hexdigest()

FD003_CLEAN_DEV_EVAL_PROTOCOL_LOCKED_629W = True

FD003_DEV_OUTCOMES_INSPECTED_629W = False
FD003_DEV_SCORING_PERFORMED_629W = False
FD003_THRESHOLD_SELECTED_629W = False
FD003_TEST_CONTENTS_READ_629W = False
FD003_RUL_CONTENTS_READ_629W = False

all_checks_passed_629w = True


# =============================================================================
# 6) Output
# =============================================================================

print("=" * 136)

print(
    "STEP 6.29W — FD003 CLEAN-DEV "
    "SELECTIVE-EVALUATION PROTOCOL FREEZE"
)

print("=" * 136)

print("DEV outcomes inspected         :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FROZEN EVALUATION PROTOCOL ---")

print(
    "Population                     : clean FD003 DEV windows"
)

print(
    "Population size                :",
    len(risk_629w),
)

print(
    "Risk orientation               : "
    "lower risk = more reliable = accepted first"
)

print(
    "Primary ordering metric        : "
    "Spearman(risk, absolute RUL error)"
)

print(
    "Selective gain                 : "
    "full DEV MAE - accepted-subset MAE"
)

print(
    "Diagnostic coverages           :",
    FD003_CLEAN_DEV_EVAL_COVERAGES_629W.tolist(),
)

print(
    "Accepted counts                :",
    FD003_CLEAN_DEV_EVAL_ACCEPT_COUNTS_629W.tolist(),
)

print(
    "Count rule                     : "
    "ceil(target_coverage * 4706)"
)

print(
    "Ordering / tie rule            : "
    "stable ascending risk / original DEV order"
)

print(
    "Nominal operating reference    :",
    0.80,
)

print(
    "Threshold selected             :",
    False,
)

print(
    "\nEvaluation-protocol SHA-256    :",
    FD003_CLEAN_DEV_EVAL_PROTOCOL_SHA256_629W,
)

print(
    "\nPASS — the clean-DEV outcome-evaluation protocol is frozen "
    "before inspecting the risk/error relationship."
)

print(
    "Next: execute this frozen protocol once using DEV outcomes only. "
    "No TEST, official RUL, corruption-policy selection, "
    "or threshold freeze will occur in that scoring step."
)

In [ ]:
# STEP 6.29X
# FD003 CLEAN-DEV FROZEN SELECTIVE-RISK OUTCOME EVALUATION
#
# Executes the prospectively frozen STEP 6.29W evaluation exactly once.
# DEV outcomes are now allowed for reporting/calibration evidence.
#
# NO threshold selection.
# NO policy selection.
# NO corruption-policy decision.
# NO model retraining / new inference / MC rerun.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629w", False)):
    raise RuntimeError(
        "STEP 6.29W must pass before STEP 6.29X."
    )

required_629x = [
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_DEV_EVAL_PROTOCOL_629W",
    "FD003_CLEAN_DEV_EVAL_PROTOCOL_SHA256_629W",
    "FD003_CLEAN_DEV_EVAL_ACCEPT_COUNTS_629W",
    "FD003_CLEAN_DEV_EVAL_COVERAGES_629W",
]

missing_629x = [
    name
    for name in required_629x
    if name not in globals()
]

if missing_629x:
    raise RuntimeError(
        "Missing required STEP 6.29X object(s): "
        + ", ".join(missing_629x)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629W", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29X."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629W", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29X."
    )

if bool(
    globals().get(
        "FD003_CLEAN_DEV_OUTCOME_EVAL_COMPLETE_629X",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29X has already been completed."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629x(value):

    arr_629x = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629x.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29X hashing."
        )

    return hashlib.sha256(
        arr_629x.tobytes(order="C")
    ).hexdigest()


def _average_ranks_629x(x):

    x_629x = np.asarray(
        x,
        dtype=np.float64,
    ).reshape(-1)

    order_629x = np.argsort(
        x_629x,
        kind="mergesort",
    )

    sorted_x_629x = x_629x[
        order_629x
    ]

    ranks_629x = np.empty(
        len(x_629x),
        dtype=np.float64,
    )

    i_629x = 0

    while i_629x < len(x_629x):

        j_629x = i_629x + 1

        while (
            j_629x < len(x_629x)
            and
            sorted_x_629x[j_629x]
            ==
            sorted_x_629x[i_629x]
        ):
            j_629x += 1

        average_rank_629x = (
            0.5
            *
            (
                (i_629x + 1)
                +
                j_629x
            )
        )

        ranks_629x[
            order_629x[
                i_629x:j_629x
            ]
        ] = average_rank_629x

        i_629x = j_629x

    return ranks_629x


# =============================================================================
# 3) Verify STEP 6.29W protocol unchanged
# =============================================================================

current_protocol_sha_629x = hashlib.sha256(
    json.dumps(
        FD003_CLEAN_DEV_EVAL_PROTOCOL_629W,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

if (
    current_protocol_sha_629x
    !=
    FD003_CLEAN_DEV_EVAL_PROTOCOL_SHA256_629W
):
    raise RuntimeError(
        "STEP 6.29W evaluation protocol changed before scoring."
    )


# =============================================================================
# 4) Verify frozen clean DEV risk unchanged
# =============================================================================

risk_629x = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
    dtype=np.float64,
).reshape(-1)

if risk_629x.shape != (4706,):
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk shape changed."
    )

if not np.isfinite(
    risk_629x
).all():
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk contains non-finite values."
    )

if (
    _raw_sha256_629x(risk_629x)
    !=
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V
):
    raise RuntimeError(
        "Frozen FD003 clean-DEV risk fingerprint changed before scoring."
    )


# =============================================================================
# 5) Frozen STEP 6.29L DEV evidence identities
#
# These hashes were established before the current reliability evaluation.
# We recover the corresponding DEV-only evidence by exact identity.
# =============================================================================

EXPECTED_DEV_PRED_SHA_629X = (
    "014a4b16d260ff2446c77f55c285309bdd07494c9332f92e74558b2543ab9568"
)

EXPECTED_DEV_TARGET_SHA_629X = (
    "160ece15b0a92ccb4ce7a83100e3c58dba6596d21e3d04871563d609d0b39c89"
)

EXPECTED_DEV_ABSERR_SHA_629X = (
    "b88d2981a9fdb1338453a83917423b174e0b54c431ffe8ff82f64d9ee8e1cb23"
)


# =============================================================================
# 6) Safe DEV-only exact-hash recovery
#
# Names are filtered BEFORE any value is inspected.
# TEST / official-RUL objects are explicitly excluded.
# =============================================================================

safe_candidate_names_629x = []

for name_629x in sorted(
    globals().keys()
):

    lower_629x = name_629x.lower()

    if "dev" not in lower_629x:
        continue

    if any(
        token_629x in lower_629x
        for token_629x in (
            "test",
            "official_rul",
            "rul_fd",
            "score_table",
        )
    ):
        continue

    if not any(
        token_629x in lower_629x
        for token_629x in (
            "pred",
            "target",
            "error",
            "abs",
        )
    ):
        continue

    safe_candidate_names_629x.append(
        name_629x
    )


matches_629x = {
    "prediction": [],
    "target": [],
    "absolute_error": [],
}


for name_629x in safe_candidate_names_629x:

    obj_629x = globals()[
        name_629x
    ]

    try:
        arr_629x = np.asarray(
            obj_629x
        )
    except Exception:
        continue

    if (
        arr_629x.dtype.hasobject
        or
        arr_629x.ndim == 0
    ):
        continue

    arr_1d_629x = np.asarray(
        arr_629x
    ).reshape(-1)

    if arr_1d_629x.shape != (4706,):
        continue

    try:
        arr64_629x = np.asarray(
            arr_1d_629x,
            dtype=np.float64,
        )
    except Exception:
        continue

    if not np.isfinite(
        arr64_629x
    ).all():
        continue

    sha_629x = _raw_sha256_629x(
        arr64_629x
    )

    if sha_629x == EXPECTED_DEV_PRED_SHA_629X:
        matches_629x[
            "prediction"
        ].append(
            name_629x
        )

    if sha_629x == EXPECTED_DEV_TARGET_SHA_629X:
        matches_629x[
            "target"
        ].append(
            name_629x
        )

    if sha_629x == EXPECTED_DEV_ABSERR_SHA_629X:
        matches_629x[
            "absolute_error"
        ].append(
            name_629x
        )


# =============================================================================
# 7) Recover frozen DEV absolute-error evidence
# =============================================================================

abs_error_629x = None
abs_error_source_629x = None


if matches_629x["absolute_error"]:

    source_name_629x = matches_629x[
        "absolute_error"
    ][0]

    abs_error_629x = np.asarray(
        globals()[
            source_name_629x
        ],
        dtype=np.float64,
    ).reshape(-1)

    abs_error_source_629x = (
        source_name_629x
    )


elif (
    matches_629x["prediction"]
    and
    matches_629x["target"]
):

    pred_name_629x = matches_629x[
        "prediction"
    ][0]

    target_name_629x = matches_629x[
        "target"
    ][0]

    prediction_629x = np.asarray(
        globals()[
            pred_name_629x
        ],
        dtype=np.float64,
    ).reshape(-1)

    target_629x = np.asarray(
        globals()[
            target_name_629x
        ],
        dtype=np.float64,
    ).reshape(-1)

    abs_error_629x = np.abs(
        prediction_629x
        -
        target_629x
    )

    if (
        _raw_sha256_629x(
            abs_error_629x
        )
        !=
        EXPECTED_DEV_ABSERR_SHA_629X
    ):
        raise RuntimeError(
            "Re-derived DEV absolute error does not match "
            "frozen STEP 6.29L evidence."
        )

    abs_error_source_629x = (
        "derived_from:"
        +
        pred_name_629x
        +
        "|"
        +
        target_name_629x
    )


else:

    raise RuntimeError(
        "Could not safely recover the frozen STEP 6.29L DEV "
        "outcome evidence by exact SHA-256. No scoring was performed."
    )


if (
    _raw_sha256_629x(
        abs_error_629x
    )
    !=
    EXPECTED_DEV_ABSERR_SHA_629X
):
    raise RuntimeError(
        "Recovered DEV absolute-error fingerprint does not "
        "match STEP 6.29L."
    )


# =============================================================================
# 8) Execute the prospectively frozen ordering metric
# =============================================================================

risk_ranks_629x = _average_ranks_629x(
    risk_629x
)

error_ranks_629x = _average_ranks_629x(
    abs_error_629x
)

spearman_629x = float(
    np.corrcoef(
        risk_ranks_629x,
        error_ranks_629x,
    )[0, 1]
)

full_dev_mae_629x = float(
    abs_error_629x.mean()
)


# =============================================================================
# 9) Execute frozen fixed-coverage diagnostics
#
# Lower risk = accepted first.
# Stable mergesort preserves original DEV order for ties.
# =============================================================================

stable_order_629x = np.argsort(
    risk_629x,
    kind="mergesort",
)

coverages_629x = np.asarray(
    FD003_CLEAN_DEV_EVAL_COVERAGES_629W,
    dtype=np.float64,
)

accept_counts_629x = np.asarray(
    FD003_CLEAN_DEV_EVAL_ACCEPT_COUNTS_629W,
    dtype=np.int64,
)

rows_629x = []
mask_hashes_629x = {}


for coverage_629x, k_629x in zip(
    coverages_629x,
    accept_counts_629x,
):

    k_629x = int(
        k_629x
    )

    accept_mask_629x = np.zeros(
        len(
            risk_629x
        ),
        dtype=bool,
    )

    accept_mask_629x[
        stable_order_629x[
            :k_629x
        ]
    ] = True

    abstain_mask_629x = (
        ~accept_mask_629x
    )

    accepted_mae_629x = float(
        abs_error_629x[
            accept_mask_629x
        ].mean()
    )

    abstained_mae_629x = float(
        abs_error_629x[
            abstain_mask_629x
        ].mean()
    )

    gain_629x = float(
        full_dev_mae_629x
        -
        accepted_mae_629x
    )

    relative_gain_pct_629x = float(
        100.0
        *
        gain_629x
        /
        full_dev_mae_629x
    )

    coverage_key_629x = (
        f"{coverage_629x:.2f}"
    )

    mask_hashes_629x[
        coverage_key_629x
    ] = _raw_sha256_629x(
        accept_mask_629x
    )

    rows_629x.append(
        {
            "target_coverage":
                float(
                    coverage_629x
                ),

            "accepted_count":
                int(
                    k_629x
                ),

            "realized_coverage":
                float(
                    k_629x
                    /
                    len(
                        risk_629x
                    )
                ),

            "full_dev_mae":
                full_dev_mae_629x,

            "accepted_dev_mae":
                accepted_mae_629x,

            "abstained_dev_mae":
                abstained_mae_629x,

            "selective_mae_gain":
                gain_629x,

            "relative_mae_reduction_pct":
                relative_gain_pct_629x,
        }
    )


FD003_CLEAN_DEV_SELECTIVE_EVAL_TABLE_629X = pd.DataFrame(
    rows_629x
)


# =============================================================================
# 10) Evidence lock
# =============================================================================

FD003_CLEAN_DEV_OUTCOME_EVIDENCE_629X = {

    "step":
        "6.29X",

    "dataset":
        "FD003",

    "evaluation_protocol_sha256":
        FD003_CLEAN_DEV_EVAL_PROTOCOL_SHA256_629W,

    "risk_sha256":
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,

    "absolute_error_sha256":
        EXPECTED_DEV_ABSERR_SHA_629X,

    "absolute_error_source":
        abs_error_source_629x,

    "population_size":
        4706,

    "full_dev_mae":
        full_dev_mae_629x,

    "spearman_risk_vs_absolute_error":
        spearman_629x,

    "diagnostic_rows":
        rows_629x,

    "accept_mask_sha256_by_coverage":
        mask_hashes_629x,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "corruption_results_used":
        False,

    "model_retrained":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


evidence_payload_629x = json.dumps(
    FD003_CLEAN_DEV_OUTCOME_EVIDENCE_629X,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_CLEAN_DEV_OUTCOME_EVIDENCE_SHA256_629X = hashlib.sha256(
    evidence_payload_629x
).hexdigest()


FD003_CLEAN_DEV_OUTCOME_EVAL_COMPLETE_629X = True

FD003_DEV_OUTCOMES_INSPECTED_629X = True
FD003_DEV_SCORING_PERFORMED_629X = True

FD003_THRESHOLD_SELECTED_629X = False
FD003_POLICY_SELECTED_629X = False

FD003_TEST_CONTENTS_READ_629X = False
FD003_RUL_CONTENTS_READ_629X = False

all_checks_passed_629x = True


# =============================================================================
# 11) Output
# =============================================================================

print("=" * 136)

print(
    "STEP 6.29X — FD003 CLEAN-DEV FROZEN "
    "SELECTIVE-RISK OUTCOME EVALUATION"
)

print("=" * 136)

print("DEV outcomes inspected         :", True)
print("DEV error scoring performed    :", True)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("Corruption results used        :", False)
print("Model retrained                :", False)
print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- OUTCOME EVIDENCE IDENTITY ---")

print(
    "Recovered absolute-error source:",
    abs_error_source_629x,
)

print(
    "Absolute-error SHA-256         :",
    _raw_sha256_629x(
        abs_error_629x
    ),
)

print(
    "Risk SHA-256                   :",
    _raw_sha256_629x(
        risk_629x
    ),
)

print(
    "Evaluation protocol SHA-256    :",
    current_protocol_sha_629x,
)

print("\n--- CLEAN DEV ORDERING EVIDENCE ---")

print(
    "Full clean DEV MAE             :",
    full_dev_mae_629x,
)

print(
    "Spearman(risk, abs error)      :",
    spearman_629x,
)

print("\n--- FROZEN COVERAGE DIAGNOSTICS ---")

print(
    FD003_CLEAN_DEV_SELECTIVE_EVAL_TABLE_629X.to_string(
        index=False
    )
)

print(
    "\nOutcome-evidence SHA-256       :",
    FD003_CLEAN_DEV_OUTCOME_EVIDENCE_SHA256_629X,
)

print(
    "\nPASS — STEP 6.29X evaluated the already-frozen clean-DEV risk "
    "exactly once under the prospectively frozen STEP 6.29W protocol."
)

print(
    "No threshold or operating policy was selected; no corruption result, "
    "TEST, official RUL, retraining, model inference, or MC rerun was used."
)

print(
    "Next: interpret this clean-DEV ordering evidence, then freeze the "
    "controlled sensor-corruption experiment protocol before executing "
    "corruption runs."
)

In [ ]:
# STEP 6.29Y
# FD003 HISTORICAL CONTROLLED-CORRUPTION CONTRACT LOCATOR
#
# Metadata-only recovery step.
# NO corruption executed.
# NO model inference / NO MC execution.
# NO threshold or policy selection.
# NO TEST / official RUL access.
# No numeric payloads from historical corruption objects are inspected here.

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_629x", False)):
    raise RuntimeError(
        "STEP 6.29X must pass before STEP 6.29Y."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629X", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29Y."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629X", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29Y."
    )

corruption_tokens_629y = (
    "corrupt",
    "noise",
    "bias",
    "drift",
    "dropout",
    "stuck",
    "freeze",
    "scenario",
)

historical_tokens_629y = (
    "fd001",
    "fd002",
    "628",
)

forbidden_tokens_629y = (
    "test",
    "official_rul",
    "rul_fd",
    "score",
    "mae",
    "error",
)

candidate_names_629y = []

for name_629y in sorted(globals().keys()):

    lower_629y = name_629y.lower()

    if any(
        token_629y in lower_629y
        for token_629y in forbidden_tokens_629y
    ):
        continue

    if not any(
        token_629y in lower_629y
        for token_629y in corruption_tokens_629y
    ):
        continue

    if not any(
        token_629y in lower_629y
        for token_629y in historical_tokens_629y
    ):
        continue

    candidate_names_629y.append(
        name_629y
    )

rows_629y = []

for name_629y in candidate_names_629y:

    obj_629y = globals()[name_629y]

    row_629y = {
        "name": name_629y,
        "type": type(obj_629y).__name__,
        "shape_or_len": None,
        "dtype": None,
        "dict_keys_or_fields": None,
        "callable_signature": None,
    }

    if isinstance(obj_629y, np.ndarray):

        row_629y["shape_or_len"] = tuple(
            int(x)
            for x in obj_629y.shape
        )

        row_629y["dtype"] = str(
            obj_629y.dtype
        )

        if obj_629y.dtype.names is not None:
            row_629y["dict_keys_or_fields"] = [
                str(x)
                for x in obj_629y.dtype.names
            ]

    elif isinstance(obj_629y, pd.DataFrame):

        row_629y["shape_or_len"] = tuple(
            int(x)
            for x in obj_629y.shape
        )

        row_629y["dict_keys_or_fields"] = [
            str(c)
            for c in obj_629y.columns
        ]

    elif isinstance(obj_629y, dict):

        row_629y["shape_or_len"] = int(
            len(obj_629y)
        )

        row_629y["dict_keys_or_fields"] = sorted(
            str(k)
            for k in obj_629y.keys()
        )

    elif isinstance(obj_629y, (list, tuple)):

        row_629y["shape_or_len"] = int(
            len(obj_629y)
        )

    elif callable(obj_629y):

        try:
            import inspect

            row_629y["callable_signature"] = str(
                inspect.signature(obj_629y)
            )

        except Exception:
            row_629y["callable_signature"] = "<unavailable>"

    else:

        try:
            row_629y["shape_or_len"] = int(
                len(obj_629y)
            )

        except Exception:
            row_629y["shape_or_len"] = "<scalar_or_uninspected>"

    rows_629y.append(
        row_629y
    )

FD003_HISTORICAL_CORRUPTION_LOCATOR_629Y = pd.DataFrame(
    rows_629y
)

FD003_CORRUPTION_EXECUTED_629Y = False
FD003_THRESHOLD_SELECTED_629Y = False
FD003_POLICY_SELECTED_629Y = False
FD003_TEST_CONTENTS_READ_629Y = False
FD003_RUL_CONTENTS_READ_629Y = False

all_checks_passed_629y = True

print("=" * 136)

print(
    "STEP 6.29Y — FD003 HISTORICAL CONTROLLED-CORRUPTION "
    "CONTRACT LOCATOR"
)

print("=" * 136)

print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- TARGETED HISTORICAL CORRUPTION OBJECT INVENTORY ---")

if len(FD003_HISTORICAL_CORRUPTION_LOCATOR_629Y) == 0:

    print(
        "<no targeted historical corruption objects found>"
    )

else:

    print(
        FD003_HISTORICAL_CORRUPTION_LOCATOR_629Y.to_string(
            index=False
        )
    )

print(
    "\nPASS — metadata-only locator complete. "
    "No historical numeric corruption payload, DEV outcome, TEST, or RUL value "
    "was inspected by this step."
)

print(
    "Next action depends on this inventory: recover the exact historical "
    "corruption scenario-generation contract before freezing FD003 corruption experiments."
)

In [ ]:
# STEP 6.29Z
# FD003 HISTORICAL CONTROLLED-CORRUPTION PROTOCOL + OPERATOR RECOVERY
#
# Read-only, strict-whitelist recovery.
# NO corruption execution.
# NO model inference / MC execution.
# NO DEV outcome/error scoring.
# NO threshold or policy selection.
# NO TEST / official RUL access.
# NO FD003 corruption protocol is frozen by this cell.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629y", False)):
    raise RuntimeError(
        "STEP 6.29Y must pass before STEP 6.29Z."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629Y", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29Z."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629Y", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29Z."
    )


required_objects_629z = [
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD002_CORRUPTION_MANIFEST_CONTRACT_628GD",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
]

required_functions_629z = [
    "apply_fd002_bias_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
]

missing_629z = [
    name
    for name in (
        required_objects_629z
        +
        required_functions_629z
    )
    if name not in globals()
]

if missing_629z:
    raise RuntimeError(
        "Missing required historical corruption object(s): "
        + ", ".join(missing_629z)
    )


# =============================================================================
# 2) Strict type / schema checks
# =============================================================================

transfer_629z = FD002_CORRUPTION_TRANSFER_CONTRACT_628GB
manifest_contract_629z = FD002_CORRUPTION_MANIFEST_CONTRACT_628GD
manifest_629z = FD002_CORRUPTION_MANIFEST_628GD
scale_ref_629z = FD002_CORRUPTION_SCALE_REFERENCE_628GB
dropout_ref_629z = FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
operator_contract_629z = FD002_CORRUPTION_OPERATOR_CONTRACT_628GC

if not isinstance(transfer_629z, dict):
    raise RuntimeError(
        "FD002 transfer contract is not a dict."
    )

if not isinstance(manifest_contract_629z, dict):
    raise RuntimeError(
        "FD002 manifest contract is not a dict."
    )

if not isinstance(operator_contract_629z, dict):
    raise RuntimeError(
        "FD002 operator contract is not a dict."
    )

if not isinstance(manifest_629z, pd.DataFrame):
    raise RuntimeError(
        "FD002 corruption manifest is not a DataFrame."
    )

if not isinstance(scale_ref_629z, pd.DataFrame):
    raise RuntimeError(
        "FD002 corruption scale reference is not a DataFrame."
    )

if not isinstance(dropout_ref_629z, pd.DataFrame):
    raise RuntimeError(
        "FD002 dropout reference is not a DataFrame."
    )


expected_manifest_columns_629z = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if list(manifest_629z.columns) != expected_manifest_columns_629z:
    raise RuntimeError(
        "Historical FD002 corruption manifest schema differs "
        "from the recovered contract."
    )


# =============================================================================
# 3) Safe JSON conversion helper
# =============================================================================

def _json_safe_629z(value):

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return [
            _json_safe_629z(x)
            for x in value.tolist()
        ]

    if isinstance(value, dict):
        return {
            str(k): _json_safe_629z(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            _json_safe_629z(x)
            for x in value
        ]

    return value


# =============================================================================
# 4) Recover exact historical contract values
# =============================================================================

transfer_safe_629z = _json_safe_629z(
    transfer_629z
)

manifest_contract_safe_629z = _json_safe_629z(
    manifest_contract_629z
)

operator_contract_safe_629z = _json_safe_629z(
    operator_contract_629z
)


# =============================================================================
# 5) Recover exact manifest-generation semantics
# =============================================================================

if len(manifest_629z) != 315:
    raise RuntimeError(
        "Historical FD002 corruption manifest is not 315 rows."
    )

if not manifest_629z["scenario_id"].is_unique:
    raise RuntimeError(
        "Historical scenario_id values are not unique."
    )

if not manifest_629z["scenario_number"].is_unique:
    raise RuntimeError(
        "Historical scenario_number values are not unique."
    )


family_order_629z = list(
    dict.fromkeys(
        manifest_629z[
            "family"
        ].tolist()
    )
)

severity_order_629z = list(
    dict.fromkeys(
        manifest_629z[
            "severity"
        ].tolist()
    )
)

sensor_order_629z = list(
    dict.fromkeys(
        manifest_629z[
            "sensor"
        ].tolist()
    )
)

direction_order_629z = list(
    dict.fromkeys(
        [
            x
            for x in manifest_629z[
                "direction"
            ].tolist()
            if pd.notna(x)
        ]
    )
)

family_counts_629z = (
    manifest_629z
    .groupby(
        "family",
        sort=False,
    )
    .size()
    .astype(int)
    .to_dict()
)

manifest_parameter_summary_629z = (
    manifest_629z
    .groupby(
        [
            "family",
            "severity",
        ],
        sort=False,
        dropna=False,
    )
    .agg(
        scenario_count=(
            "scenario_id",
            "size",
        ),
        alpha_values=(
            "alpha",
            lambda s: sorted(
                {
                    float(x)
                    for x in s.dropna()
                }
            ),
        ),
        direction_values=(
            "direction",
            lambda s: list(
                dict.fromkeys(
                    [
                        str(x)
                        for x in s.dropna()
                    ]
                )
            ),
        ),
        requested_fraction_values=(
            "requested_fraction",
            lambda s: sorted(
                {
                    float(x)
                    for x in s.dropna()
                }
            ),
        ),
        seed_values=(
            "seed",
            lambda s: sorted(
                {
                    int(x)
                    for x in s.dropna()
                }
            ),
        ),
    )
    .reset_index()
)


# =============================================================================
# 6) Exact historical reference-table audits
# =============================================================================

expected_scale_cols_629z = [
    "sensor",
    "pooled_within_regime_std",
    "global_raw_std",
    "global_to_within_ratio",
]

if list(scale_ref_629z.columns) != expected_scale_cols_629z:
    raise RuntimeError(
        "Historical scale-reference schema differs from expected."
    )

expected_dropout_cols_629z = [
    "regime_id",
    "sensor",
    "mean",
]

if list(dropout_ref_629z.columns) != expected_dropout_cols_629z:
    raise RuntimeError(
        "Historical dropout-reference schema differs from expected."
    )

scale_ref_summary_629z = {
    "rows": int(len(scale_ref_629z)),
    "sensors": scale_ref_629z["sensor"].astype(str).tolist(),
    "finite_pooled_within_regime_std": bool(
        np.isfinite(
            scale_ref_629z[
                "pooled_within_regime_std"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
    "finite_global_raw_std": bool(
        np.isfinite(
            scale_ref_629z[
                "global_raw_std"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
}

dropout_ref_summary_629z = {
    "rows": int(len(dropout_ref_629z)),
    "regimes": sorted(
        int(x)
        for x in dropout_ref_629z[
            "regime_id"
        ].unique()
    ),
    "sensors": list(
        dict.fromkeys(
            dropout_ref_629z[
                "sensor"
            ].astype(str)
            .tolist()
        )
    ),
    "means_finite": bool(
        np.isfinite(
            dropout_ref_629z[
                "mean"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
}


# =============================================================================
# 7) Recover exact whitelisted operator implementations
# =============================================================================

operator_rows_629z = []
operator_sources_629z = {}

for name_629z in required_functions_629z:

    fn_629z = globals()[
        name_629z
    ]

    if not callable(fn_629z):
        raise RuntimeError(
            f"{name_629z} is not callable."
        )

    signature_629z = str(
        inspect.signature(
            fn_629z
        )
    )

    try:
        source_629z = inspect.getsource(
            fn_629z
        )
    except Exception:
        source_629z = None

    source_sha_629z = (
        hashlib.sha256(
            source_629z.encode(
                "utf-8"
            )
        ).hexdigest()
        if source_629z is not None
        else None
    )

    code_629z = getattr(
        fn_629z,
        "__code__",
        None,
    )

    bytecode_sha_629z = (
        hashlib.sha256(
            code_629z.co_code
        ).hexdigest()
        if code_629z is not None
        else None
    )

    operator_rows_629z.append(
        {
            "name":
                name_629z,

            "signature":
                signature_629z,

            "source_available":
                source_629z is not None,

            "source_sha256":
                source_sha_629z,

            "bytecode_sha256":
                bytecode_sha_629z,
        }
    )

    operator_sources_629z[
        name_629z
    ] = source_629z


FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z = pd.DataFrame(
    operator_rows_629z
)


# =============================================================================
# 8) Structural fingerprints
# =============================================================================

manifest_records_629z = []

for row_629z in manifest_629z[
    expected_manifest_columns_629z
].itertuples(
    index=False,
    name=None,
):

    manifest_records_629z.append(
        [
            None
            if pd.isna(x)
            else _json_safe_629z(x)
            for x in row_629z
        ]
    )


manifest_sha_629z = hashlib.sha256(
    json.dumps(
        manifest_records_629z,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )
).hexdigest()


recovery_payload_629z = {
    "transfer_contract":
        transfer_safe_629z,

    "manifest_contract":
        manifest_contract_safe_629z,

    "operator_contract":
        operator_contract_safe_629z,

    "manifest_sha256":
        manifest_sha_629z,

    "manifest_rows":
        int(
            len(
                manifest_629z
            )
        ),

    "family_order":
        family_order_629z,

    "severity_order":
        severity_order_629z,

    "sensor_order":
        sensor_order_629z,

    "direction_order":
        direction_order_629z,

    "family_counts":
        {
            str(k): int(v)
            for k, v in family_counts_629z.items()
        },

    "scale_reference_summary":
        scale_ref_summary_629z,

    "dropout_reference_summary":
        dropout_ref_summary_629z,

    "operator_source_sha256":
        {
            row["name"]:
                row["source_sha256"]
            for row in operator_rows_629z
        },
}


FD003_HISTORICAL_CORRUPTION_RECOVERY_SHA256_629Z = hashlib.sha256(
    json.dumps(
        recovery_payload_629z,
        sort_keys=True,
        separators=(",", ":"),
    ).encode(
        "utf-8"
    )
).hexdigest()


FD003_HISTORICAL_CORRUPTION_RECOVERY_629Z = recovery_payload_629z

FD003_CORRUPTION_EXECUTED_629Z = False
FD003_THRESHOLD_SELECTED_629Z = False
FD003_POLICY_SELECTED_629Z = False
FD003_TEST_CONTENTS_READ_629Z = False
FD003_RUL_CONTENTS_READ_629Z = False

all_checks_passed_629z = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 140)

print(
    "STEP 6.29Z — FD003 HISTORICAL CONTROLLED-CORRUPTION "
    "PROTOCOL + OPERATOR RECOVERY"
)

print("=" * 140)

print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV outcome/error scoring      :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FD002 CORRUPTION TRANSFER CONTRACT ---"
)

print(
    json.dumps(
        transfer_safe_629z,
        indent=2,
        sort_keys=True,
    )
)

print(
    "\n--- FD002 CORRUPTION MANIFEST CONTRACT ---"
)

print(
    json.dumps(
        manifest_contract_safe_629z,
        indent=2,
        sort_keys=True,
    )
)

print(
    "\n--- FD002 CORRUPTION OPERATOR CONTRACT ---"
)

print(
    json.dumps(
        operator_contract_safe_629z,
        indent=2,
        sort_keys=True,
    )
)

print(
    "\n--- MANIFEST ORDER / COUNTS ---"
)

print(
    "Manifest rows                  :",
    len(
        manifest_629z
    ),
)

print(
    "Manifest SHA-256               :",
    manifest_sha_629z,
)

print(
    "Family order                   :",
    family_order_629z,
)

print(
    "Severity order                 :",
    severity_order_629z,
)

print(
    "Sensor order                   :",
    sensor_order_629z,
)

print(
    "Direction order                :",
    direction_order_629z,
)

print(
    "Scenario counts by family      :",
    family_counts_629z,
)

print(
    "\n--- FAMILY / SEVERITY PARAMETER SUMMARY ---"
)

print(
    manifest_parameter_summary_629z.to_string(
        index=False
    )
)

print(
    "\n--- FIT-DERIVED REFERENCE STRUCTURE ---"
)

print(
    "Scale reference summary        :",
    scale_ref_summary_629z,
)

print(
    "Dropout reference summary      :",
    dropout_ref_summary_629z,
)

print(
    "\n--- HISTORICAL OPERATOR IMPLEMENTATION AUDIT ---"
)

print(
    FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z.to_string(
        index=False
    )
)

print(
    "\nHistorical corruption recovery SHA-256 :",
    FD003_HISTORICAL_CORRUPTION_RECOVERY_SHA256_629Z,
)

print(
    "\nPASS — exact historical scenario-generation contracts, manifest "
    "semantics, FIT-derived reference structure, and operator implementations "
    "were recovered from a strict whitelist."
)

print(
    "No corruption, DEV outcome/error scoring, threshold/policy selection, "
    "model inference, MC execution, TEST, or official RUL access occurred."
)

print(
    "Next: use only this recovered methodology to define and freeze the "
    "FD003-specific controlled-corruption protocol and manifest. "
    "FD002 dataset-specific scale/imputation values will not be reused."
)

In [ ]:
# STEP 6.29Z-R1
# FD003 HISTORICAL CORRUPTION RECOVERY — JSON-SAFE PROVENANCE REPAIR
#
# Read-only repair of STEP 6.29Z fingerprint serialization.
# NO corruption / inference / MC / scoring / threshold / policy / TEST / RUL.

import hashlib
import inspect
import json
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions: use only the exact historical whitelist
# =============================================================================

if not bool(globals().get("all_checks_passed_629y", False)):
    raise RuntimeError(
        "STEP 6.29Y must pass before STEP 6.29Z-R1."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629Y", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29Z-R1."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629Y", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29Z-R1."
    )

required_objects_629zr1 = [
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD002_CORRUPTION_MANIFEST_CONTRACT_628GD",
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
]

required_functions_629zr1 = [
    "apply_fd002_bias_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
]

missing_629zr1 = [
    name
    for name in (
        required_objects_629zr1
        +
        required_functions_629zr1
    )
    if name not in globals()
]

if missing_629zr1:
    raise RuntimeError(
        "Missing required historical corruption object(s): "
        + ", ".join(missing_629zr1)
    )


transfer_629zr1 = FD002_CORRUPTION_TRANSFER_CONTRACT_628GB
manifest_contract_629zr1 = FD002_CORRUPTION_MANIFEST_CONTRACT_628GD
manifest_629zr1 = FD002_CORRUPTION_MANIFEST_628GD
scale_ref_629zr1 = FD002_CORRUPTION_SCALE_REFERENCE_628GB
dropout_ref_629zr1 = FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
operator_contract_629zr1 = FD002_CORRUPTION_OPERATOR_CONTRACT_628GC


# =============================================================================
# 2) Exact schema checks
# =============================================================================

if not isinstance(transfer_629zr1, dict):
    raise RuntimeError("Historical transfer contract is not a dict.")

if not isinstance(manifest_contract_629zr1, dict):
    raise RuntimeError("Historical manifest contract is not a dict.")

if not isinstance(operator_contract_629zr1, dict):
    raise RuntimeError("Historical operator contract is not a dict.")

if not isinstance(manifest_629zr1, pd.DataFrame):
    raise RuntimeError("Historical corruption manifest is not a DataFrame.")

if not isinstance(scale_ref_629zr1, pd.DataFrame):
    raise RuntimeError("Historical scale reference is not a DataFrame.")

if not isinstance(dropout_ref_629zr1, pd.DataFrame):
    raise RuntimeError("Historical dropout reference is not a DataFrame.")

expected_manifest_columns_629zr1 = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if list(manifest_629zr1.columns) != expected_manifest_columns_629zr1:
    raise RuntimeError(
        "Historical corruption manifest schema differs from expected."
    )

if len(manifest_629zr1) != 315:
    raise RuntimeError(
        "Historical FD002 corruption manifest is not 315 rows."
    )

if not manifest_629zr1["scenario_id"].is_unique:
    raise RuntimeError(
        "Historical scenario_id values are not unique."
    )

if not manifest_629zr1["scenario_number"].is_unique:
    raise RuntimeError(
        "Historical scenario_number values are not unique."
    )


# =============================================================================
# 3) Deterministic JSON-safe canonicalizer
# =============================================================================

def _callable_descriptor_629zr1(fn):
    try:
        signature = str(inspect.signature(fn))
    except Exception:
        signature = "<unavailable>"

    try:
        source = inspect.getsource(fn)
    except Exception:
        source = None

    code = getattr(fn, "__code__", None)

    return {
        "kind": "callable",
        "module": str(getattr(fn, "__module__", "")),
        "qualname": str(
            getattr(
                fn,
                "__qualname__",
                getattr(fn, "__name__", type(fn).__name__),
            )
        ),
        "signature": signature,
        "source_sha256": (
            hashlib.sha256(
                source.encode("utf-8")
            ).hexdigest()
            if source is not None
            else None
        ),
        "bytecode_sha256": (
            hashlib.sha256(
                code.co_code
            ).hexdigest()
            if code is not None
            else None
        ),
    }


def _json_safe_629zr1(value):

    if value is None:
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    if isinstance(value, np.generic):
        return _json_safe_629zr1(
            value.item()
        )

    if isinstance(value, Path):
        return str(value)

    if callable(value):
        return _callable_descriptor_629zr1(
            value
        )

    if isinstance(value, np.ndarray):
        return _json_safe_629zr1(
            value.tolist()
        )

    if isinstance(value, dict):
        return {
            str(k): _json_safe_629zr1(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            _json_safe_629zr1(x)
            for x in value
        ]

    if isinstance(value, set):
        return sorted(
            [
                _json_safe_629zr1(x)
                for x in value
            ],
            key=lambda x: json.dumps(
                x,
                sort_keys=True,
                separators=(",", ":"),
            ),
        )

    if isinstance(value, pd.Series):
        return {
            "kind": "Series",
            "name": (
                None
                if value.name is None
                else str(value.name)
            ),
            "length": int(len(value)),
            "dtype": str(value.dtype),
        }

    if isinstance(value, pd.DataFrame):
        return {
            "kind": "DataFrame",
            "shape": [
                int(value.shape[0]),
                int(value.shape[1]),
            ],
            "columns": [
                str(c)
                for c in value.columns
            ],
        }

    raise TypeError(
        "Unsupported provenance value type in STEP 6.29Z-R1: "
        f"{type(value).__name__}"
    )


# =============================================================================
# 4) Recover exact historical contract values safely
# =============================================================================

transfer_safe_629zr1 = _json_safe_629zr1(
    transfer_629zr1
)

manifest_contract_safe_629zr1 = _json_safe_629zr1(
    manifest_contract_629zr1
)

operator_contract_safe_629zr1 = _json_safe_629zr1(
    operator_contract_629zr1
)

for label_629zr1, obj_629zr1 in [
    ("transfer_contract", transfer_safe_629zr1),
    ("manifest_contract", manifest_contract_safe_629zr1),
    ("operator_contract", operator_contract_safe_629zr1),
]:
    try:
        json.dumps(
            obj_629zr1,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )
    except Exception as exc:
        raise RuntimeError(
            f"{label_629zr1} remains non-serializable."
        ) from exc


# =============================================================================
# 5) Manifest-generation semantics
# =============================================================================

family_order_629zr1 = list(
    dict.fromkeys(
        manifest_629zr1["family"].tolist()
    )
)

severity_order_629zr1 = list(
    dict.fromkeys(
        manifest_629zr1["severity"].tolist()
    )
)

sensor_order_629zr1 = list(
    dict.fromkeys(
        manifest_629zr1["sensor"].tolist()
    )
)

direction_order_629zr1 = list(
    dict.fromkeys(
        [
            x
            for x in manifest_629zr1[
                "direction"
            ].tolist()
            if pd.notna(x)
        ]
    )
)

family_counts_629zr1 = (
    manifest_629zr1
    .groupby(
        "family",
        sort=False,
    )
    .size()
    .astype(int)
    .to_dict()
)

manifest_parameter_summary_629zr1 = (
    manifest_629zr1
    .groupby(
        [
            "family",
            "severity",
        ],
        sort=False,
        dropna=False,
    )
    .agg(
        scenario_count=(
            "scenario_id",
            "size",
        ),
        alpha_values=(
            "alpha",
            lambda s: sorted(
                {
                    float(x)
                    for x in s.dropna()
                }
            ),
        ),
        direction_values=(
            "direction",
            lambda s: list(
                dict.fromkeys(
                    [
                        str(x)
                        for x in s.dropna()
                    ]
                )
            ),
        ),
        requested_fraction_values=(
            "requested_fraction",
            lambda s: sorted(
                {
                    float(x)
                    for x in s.dropna()
                }
            ),
        ),
        seed_values=(
            "seed",
            lambda s: sorted(
                {
                    int(x)
                    for x in s.dropna()
                }
            ),
        ),
    )
    .reset_index()
)


# =============================================================================
# 6) FIT-derived historical reference structure
# =============================================================================

expected_scale_cols_629zr1 = [
    "sensor",
    "pooled_within_regime_std",
    "global_raw_std",
    "global_to_within_ratio",
]

if list(scale_ref_629zr1.columns) != expected_scale_cols_629zr1:
    raise RuntimeError(
        "Historical scale-reference schema differs from expected."
    )

expected_dropout_cols_629zr1 = [
    "regime_id",
    "sensor",
    "mean",
]

if list(dropout_ref_629zr1.columns) != expected_dropout_cols_629zr1:
    raise RuntimeError(
        "Historical dropout-reference schema differs from expected."
    )

scale_ref_summary_629zr1 = {
    "rows": int(len(scale_ref_629zr1)),
    "sensors": scale_ref_629zr1[
        "sensor"
    ].astype(str).tolist(),
    "finite_pooled_within_regime_std": bool(
        np.isfinite(
            scale_ref_629zr1[
                "pooled_within_regime_std"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
    "finite_global_raw_std": bool(
        np.isfinite(
            scale_ref_629zr1[
                "global_raw_std"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
}

dropout_ref_summary_629zr1 = {
    "rows": int(len(dropout_ref_629zr1)),
    "regimes": sorted(
        int(x)
        for x in dropout_ref_629zr1[
            "regime_id"
        ].unique()
    ),
    "sensors": list(
        dict.fromkeys(
            dropout_ref_629zr1[
                "sensor"
            ].astype(str).tolist()
        )
    ),
    "means_finite": bool(
        np.isfinite(
            dropout_ref_629zr1[
                "mean"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    ),
}


# =============================================================================
# 7) Historical operator structural audit
# =============================================================================

operator_rows_629zr1 = []

for name_629zr1 in required_functions_629zr1:

    fn_629zr1 = globals()[
        name_629zr1
    ]

    descriptor_629zr1 = _callable_descriptor_629zr1(
        fn_629zr1
    )

    operator_rows_629zr1.append(
        {
            "name": name_629zr1,
            "signature": descriptor_629zr1[
                "signature"
            ],
            "source_available": (
                descriptor_629zr1[
                    "source_sha256"
                ] is not None
            ),
            "source_sha256": descriptor_629zr1[
                "source_sha256"
            ],
            "bytecode_sha256": descriptor_629zr1[
                "bytecode_sha256"
            ],
        }
    )

FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z = pd.DataFrame(
    operator_rows_629zr1
)


# =============================================================================
# 8) Deterministic manifest + recovery fingerprints
# =============================================================================

def _manifest_scalar_629zr1(x):

    if pd.isna(x):
        return None

    return _json_safe_629zr1(x)


manifest_records_629zr1 = [
    [
        _manifest_scalar_629zr1(x)
        for x in row_629zr1
    ]
    for row_629zr1 in manifest_629zr1[
        expected_manifest_columns_629zr1
    ].itertuples(
        index=False,
        name=None,
    )
]

manifest_sha_629zr1 = hashlib.sha256(
    json.dumps(
        manifest_records_629zr1,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode(
        "utf-8"
    )
).hexdigest()


recovery_payload_629zr1 = {
    "transfer_contract":
        transfer_safe_629zr1,

    "manifest_contract":
        manifest_contract_safe_629zr1,

    "operator_contract":
        operator_contract_safe_629zr1,

    "manifest_sha256":
        manifest_sha_629zr1,

    "manifest_rows":
        int(
            len(
                manifest_629zr1
            )
        ),

    "family_order":
        _json_safe_629zr1(
            family_order_629zr1
        ),

    "severity_order":
        _json_safe_629zr1(
            severity_order_629zr1
        ),

    "sensor_order":
        _json_safe_629zr1(
            sensor_order_629zr1
        ),

    "direction_order":
        _json_safe_629zr1(
            direction_order_629zr1
        ),

    "family_counts":
        {
            str(k): int(v)
            for k, v in family_counts_629zr1.items()
        },

    "scale_reference_summary":
        _json_safe_629zr1(
            scale_ref_summary_629zr1
        ),

    "dropout_reference_summary":
        _json_safe_629zr1(
            dropout_ref_summary_629zr1
        ),

    "operator_source_sha256":
        {
            row["name"]:
                row["source_sha256"]
            for row in operator_rows_629zr1
        },
}


recovery_json_629zr1 = json.dumps(
    recovery_payload_629zr1,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
    allow_nan=False,
)

FD003_HISTORICAL_CORRUPTION_RECOVERY_SHA256_629Z = hashlib.sha256(
    recovery_json_629zr1.encode(
        "utf-8"
    )
).hexdigest()

FD003_HISTORICAL_CORRUPTION_RECOVERY_629Z = recovery_payload_629zr1

FD003_CORRUPTION_EXECUTED_629Z = False
FD003_THRESHOLD_SELECTED_629Z = False
FD003_POLICY_SELECTED_629Z = False
FD003_TEST_CONTENTS_READ_629Z = False
FD003_RUL_CONTENTS_READ_629Z = False

all_checks_passed_629z = True
all_checks_passed_629zr1 = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 142)
print(
    "STEP 6.29Z-R1 — FD003 HISTORICAL CORRUPTION RECOVERY "
    "JSON-SAFE PROVENANCE REPAIR"
)
print("=" * 142)

print("Original STEP 6.29Z failure cause : callable not JSON serializable")
print("Scientific execution repeated     : False")
print("Corruption executed               : False")
print("Model inference performed         : False")
print("MC-dropout execution performed    : False")
print("DEV outcome/error scoring         : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print(
    "\n--- FD002 CORRUPTION TRANSFER CONTRACT ---"
)
print(
    json.dumps(
        transfer_safe_629zr1,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
)

print(
    "\n--- FD002 CORRUPTION MANIFEST CONTRACT ---"
)
print(
    json.dumps(
        manifest_contract_safe_629zr1,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
)

print(
    "\n--- FD002 CORRUPTION OPERATOR CONTRACT "
    "(CALLABLES STRUCTURALLY REPRESENTED) ---"
)
print(
    json.dumps(
        operator_contract_safe_629zr1,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
)

print(
    "\n--- MANIFEST ORDER / COUNTS ---"
)
print("Manifest rows                  :", len(manifest_629zr1))
print("Manifest SHA-256               :", manifest_sha_629zr1)
print("Family order                   :", family_order_629zr1)
print("Severity order                 :", severity_order_629zr1)
print("Sensor order                   :", sensor_order_629zr1)
print("Direction order                :", direction_order_629zr1)
print("Scenario counts by family      :", family_counts_629zr1)

print(
    "\n--- FAMILY / SEVERITY PARAMETER SUMMARY ---"
)
print(
    manifest_parameter_summary_629zr1.to_string(
        index=False
    )
)

print(
    "\n--- FIT-DERIVED REFERENCE STRUCTURE ---"
)
print(
    "Scale reference summary        :",
    scale_ref_summary_629zr1,
)
print(
    "Dropout reference summary      :",
    dropout_ref_summary_629zr1,
)

print(
    "\n--- HISTORICAL OPERATOR IMPLEMENTATION AUDIT ---"
)
print(
    FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z.to_string(
        index=False
    )
)

print(
    "\nHistorical corruption recovery SHA-256 :",
    FD003_HISTORICAL_CORRUPTION_RECOVERY_SHA256_629Z,
)

print(
    "\nPASS — STEP 6.29Z recovery is now complete. "
    "The previous failure was limited to JSON serialization of callable "
    "metadata and did not affect scientific state."
)

print(
    "Next: review this recovered protocol, then freeze an FD003-specific "
    "controlled-corruption manifest using FD003 FIT-derived scale/imputation "
    "references. No FD002 numeric calibration values will be transferred."
)

In [ ]:
# STEP 6.29Z-R2
# FD003 HISTORICAL CORRUPTION OPERATOR-SEMANTICS RECOVERY
#
# Strict-whitelist source inspection only.
# NO corruption execution.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import inspect
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629z", False)):
    raise RuntimeError(
        "STEP 6.29Z-R1 must complete before STEP 6.29Z-R2."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629Z", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29Z-R2."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629Z", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29Z-R2."
    )

required_operators_629zr2 = [
    "apply_fd002_bias_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
]

missing_operators_629zr2 = [
    name
    for name in required_operators_629zr2
    if name not in globals()
]

if missing_operators_629zr2:
    raise RuntimeError(
        "Missing required historical operator(s): "
        + ", ".join(missing_operators_629zr2)
    )

if "FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z" not in globals():
    raise RuntimeError(
        "Historical operator audit from STEP 6.29Z-R1 is missing."
    )


# =============================================================================
# 2) Recover and verify exact sources
# =============================================================================

audit_629zr2 = (
    FD003_HISTORICAL_CORRUPTION_OPERATOR_AUDIT_629Z
    .copy()
    .set_index("name")
)

rows_629zr2 = []
sources_629zr2 = {}

for name_629zr2 in required_operators_629zr2:

    fn_629zr2 = globals()[name_629zr2]

    if not callable(fn_629zr2):
        raise RuntimeError(
            f"{name_629zr2} is not callable."
        )

    try:
        source_629zr2 = inspect.getsource(
            fn_629zr2
        )
    except Exception as exc:
        raise RuntimeError(
            f"Source unavailable for {name_629zr2}."
        ) from exc

    signature_629zr2 = str(
        inspect.signature(
            fn_629zr2
        )
    )

    source_sha_629zr2 = hashlib.sha256(
        source_629zr2.encode("utf-8")
    ).hexdigest()

    code_629zr2 = getattr(
        fn_629zr2,
        "__code__",
        None,
    )

    if code_629zr2 is None:
        raise RuntimeError(
            f"No Python code object for {name_629zr2}."
        )

    bytecode_sha_629zr2 = hashlib.sha256(
        code_629zr2.co_code
    ).hexdigest()

    if name_629zr2 not in audit_629zr2.index:
        raise RuntimeError(
            f"{name_629zr2} missing from recovered operator audit."
        )

    expected_source_sha_629zr2 = audit_629zr2.loc[
        name_629zr2,
        "source_sha256",
    ]

    expected_bytecode_sha_629zr2 = audit_629zr2.loc[
        name_629zr2,
        "bytecode_sha256",
    ]

    source_exact_629zr2 = bool(
        source_sha_629zr2
        ==
        expected_source_sha_629zr2
    )

    bytecode_exact_629zr2 = bool(
        bytecode_sha_629zr2
        ==
        expected_bytecode_sha_629zr2
    )

    if not source_exact_629zr2:
        raise RuntimeError(
            f"Source hash mismatch for {name_629zr2}."
        )

    if not bytecode_exact_629zr2:
        raise RuntimeError(
            f"Bytecode hash mismatch for {name_629zr2}."
        )

    referenced_names_629zr2 = [
        str(x)
        for x in code_629zr2.co_names
    ]

    rows_629zr2.append(
        {
            "name": name_629zr2,
            "signature": signature_629zr2,
            "source_sha256": source_sha_629zr2,
            "source_exact": source_exact_629zr2,
            "bytecode_sha256": bytecode_sha_629zr2,
            "bytecode_exact": bytecode_exact_629zr2,
            "referenced_names": referenced_names_629zr2,
        }
    )

    sources_629zr2[
        name_629zr2
    ] = source_629zr2


FD003_HISTORICAL_CORRUPTION_OPERATOR_SEMANTICS_AUDIT_629ZR2 = pd.DataFrame(
    rows_629zr2
)

FD003_HISTORICAL_CORRUPTION_OPERATOR_SOURCES_629ZR2 = dict(
    sources_629zr2
)


# =============================================================================
# 3) Lock recovery-state flags only
# =============================================================================

FD003_CORRUPTION_EXECUTED_629ZR2 = False
FD003_DEV_SCORING_PERFORMED_629ZR2 = False
FD003_THRESHOLD_SELECTED_629ZR2 = False
FD003_POLICY_SELECTED_629ZR2 = False
FD003_TEST_CONTENTS_READ_629ZR2 = False
FD003_RUL_CONTENTS_READ_629ZR2 = False

all_checks_passed_629zr2 = True


# =============================================================================
# 4) Output
# =============================================================================

print("=" * 142)

print(
    "STEP 6.29Z-R2 — FD003 HISTORICAL CONTROLLED-CORRUPTION "
    "OPERATOR-SEMANTICS RECOVERY"
)

print("=" * 142)

print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- EXACT HISTORICAL OPERATOR AUDIT ---"
)

print(
    FD003_HISTORICAL_CORRUPTION_OPERATOR_SEMANTICS_AUDIT_629ZR2.to_string(
        index=False
    )
)

print(
    "\n--- EXACT HISTORICAL OPERATOR SOURCES ---"
)

for name_629zr2 in required_operators_629zr2:

    print(
        "\n"
        + "-" * 118
    )

    print(
        name_629zr2
    )

    print(
        "-" * 118
    )

    print(
        FD003_HISTORICAL_CORRUPTION_OPERATOR_SOURCES_629ZR2[
            name_629zr2
        ]
    )

print(
    "\nPASS — exact historical corruption operator semantics were "
    "recovered and matched the previously fingerprinted implementations."
)

print(
    "No FD003 corruption, model/MC execution, DEV scoring, threshold/policy "
    "selection, TEST, or official RUL access occurred."
)

print(
    "Next: define and freeze the FD003-specific corruption protocol and "
    "315-scenario manifest using these exact operator semantics and "
    "FD003 FIT-only reference statistics."
)

In [ ]:
# STEP 6.29AA
# FD003 STRICT-WHITELIST FIT-RAW / CORRUPTION-REFERENCE SOURCE LOCATOR
#
# Read-only structural locator.
# NO corruption execution.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.
# NO corruption reference or manifest is created here.

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_629zr2", False)):
    raise RuntimeError(
        "STEP 6.29Z-R2 must pass before STEP 6.29AA."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629ZR2", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AA."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629ZR2", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AA."
    )

candidate_names_629aa = [
    "FD003_TRAIN_RAW_629B",
    "FD003_RAW_TRAIN_629B",
    "TRAIN_RAW_FD003_629B",
    "train_raw_629b",

    "FD003_FIT_RAW_629C",
    "FIT_RAW_FD003_629C",
    "fit_raw_629c",
    "fit_df_629c",
    "FD003_FIT_DF_629C",
    "FIT_DF_FD003_629C",

    "FD003_FIT_RAW_629D",
    "FIT_RAW_FD003_629D",
    "fit_raw_629d",
    "fit_df_629d",
    "FD003_FIT_DF_629D",
    "FD003_FIT_DATA_629D",

    "FIT_ENGINE_IDS_FD003_629C",
    "FD003_FIT_ENGINE_IDS_629C",
    "FIT_ENGINE_IDS_629C",

    "FD003_SELECTED_RAW_FEATURES_629E",
    "SELECTED_RAW_FEATURES_FD003_629E",
    "FD003_FEATURE_CONTRACT_629E",
    "FD003_PREPROCESSING_CONTRACT_629E",
]

rows_629aa = []

for name_629aa in candidate_names_629aa:

    present_629aa = (
        name_629aa in globals()
    )

    row_629aa = {
        "name": name_629aa,
        "present": present_629aa,
        "type": None,
        "shape_or_len": None,
        "columns_or_keys": None,
    }

    if present_629aa:

        obj_629aa = globals()[
            name_629aa
        ]

        row_629aa["type"] = (
            type(
                obj_629aa
            ).__name__
        )

        if isinstance(
            obj_629aa,
            pd.DataFrame,
        ):

            row_629aa[
                "shape_or_len"
            ] = tuple(
                int(x)
                for x in obj_629aa.shape
            )

            row_629aa[
                "columns_or_keys"
            ] = [
                str(c)
                for c in obj_629aa.columns
            ]

        elif isinstance(
            obj_629aa,
            dict,
        ):

            row_629aa[
                "shape_or_len"
            ] = int(
                len(
                    obj_629aa
                )
            )

            row_629aa[
                "columns_or_keys"
            ] = sorted(
                str(k)
                for k in obj_629aa.keys()
            )

        elif isinstance(
            obj_629aa,
            np.ndarray,
        ):

            row_629aa[
                "shape_or_len"
            ] = tuple(
                int(x)
                for x in obj_629aa.shape
            )

        elif isinstance(
            obj_629aa,
            (
                list,
                tuple,
                set,
            ),
        ):

            row_629aa[
                "shape_or_len"
            ] = int(
                len(
                    obj_629aa
                )
            )

        else:

            try:
                row_629aa[
                    "shape_or_len"
                ] = int(
                    len(
                        obj_629aa
                    )
                )

            except Exception:
                row_629aa[
                    "shape_or_len"
                ] = "<scalar_or_uninspected>"

    rows_629aa.append(
        row_629aa
    )

FD003_FIT_SOURCE_LOCATOR_629AA = (
    pd.DataFrame(
        rows_629aa
    )
)

present_table_629aa = (
    FD003_FIT_SOURCE_LOCATOR_629AA[
        FD003_FIT_SOURCE_LOCATOR_629AA[
            "present"
        ]
    ]
    .reset_index(
        drop=True
    )
)

FD003_CORRUPTION_EXECUTED_629AA = False
FD003_DEV_SCORING_PERFORMED_629AA = False
FD003_THRESHOLD_SELECTED_629AA = False
FD003_POLICY_SELECTED_629AA = False
FD003_TEST_CONTENTS_READ_629AA = False
FD003_RUL_CONTENTS_READ_629AA = False

all_checks_passed_629aa = True

print("=" * 140)

print(
    "STEP 6.29AA — FD003 STRICT-WHITELIST FIT-RAW / "
    "CORRUPTION-REFERENCE SOURCE LOCATOR"
)

print("=" * 140)

print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- PRESENT WHITELISTED FD003 FIT-SOURCE OBJECTS ---"
)

if len(
    present_table_629aa
) == 0:

    print(
        "<none of the strict-whitelist candidates are present>"
    )

else:

    print(
        present_table_629aa.to_string(
            index=False
        )
    )

print(
    "\nPASS — locator completed without inspecting TEST/RUL, "
    "DEV outcomes, or executing corruption."
)

print(
    "Next: resolve the authoritative FIT raw source and then "
    "freeze FD003 FIT-only corruption scales/imputation references "
    "plus the FD003 corruption manifest."
)

In [ ]:
# STEP 6.29AB
# FD003 AUTHORITATIVE FIT-RAW IDENTITY + FEATURE-CONTRACT LOCK
#
# Read-only provenance audit.
# NO corruption execution.
# NO corruption reference statistics constructed.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aa", False)):
    raise RuntimeError(
        "STEP 6.29AA must pass before STEP 6.29AB."
    )

required_629ab = [
    "FD003_FIT_RAW_629C",
    "fit_raw_629d",
    "FD003_FIT_ENGINE_IDS_629C",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_PREPROCESSING_CONTRACT_629E",
]

missing_629ab = [
    name
    for name in required_629ab
    if name not in globals()
]

if missing_629ab:
    raise RuntimeError(
        "Missing required STEP 6.29AB object(s): "
        + ", ".join(missing_629ab)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AA", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AB."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AA", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AB."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629ab(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused by raw hash helper."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _dataframe_sha256_629ab(df):

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            "Expected DataFrame."
        )

    hasher = hashlib.sha256()

    columns_payload = json.dumps(
        [
            str(c)
            for c in df.columns
        ],
        separators=(",", ":"),
    ).encode("utf-8")

    dtypes_payload = json.dumps(
        [
            str(x)
            for x in df.dtypes
        ],
        separators=(",", ":"),
    ).encode("utf-8")

    index_values = np.asarray(
        df.index.to_numpy()
    )

    hasher.update(
        columns_payload
    )

    hasher.update(
        dtypes_payload
    )

    if not index_values.dtype.hasobject:
        hasher.update(
            np.ascontiguousarray(
                index_values
            ).tobytes(order="C")
        )
    else:
        hasher.update(
            json.dumps(
                [
                    str(x)
                    for x in index_values
                ],
                separators=(",", ":"),
            ).encode("utf-8")
        )

    for column in df.columns:

        series = df[column]

        if series.dtype == object:

            payload = json.dumps(
                [
                    None
                    if pd.isna(x)
                    else str(x)
                    for x in series.tolist()
                ],
                separators=(",", ":"),
            ).encode("utf-8")

            hasher.update(
                payload
            )

        else:

            values = np.ascontiguousarray(
                series.to_numpy()
            )

            hasher.update(
                values.tobytes(order="C")
            )

    return hasher.hexdigest()


# =============================================================================
# 3) Strict type and schema checks
# =============================================================================

fit_c_629ab = FD003_FIT_RAW_629C
fit_d_629ab = fit_raw_629d

if not isinstance(
    fit_c_629ab,
    pd.DataFrame,
):
    raise RuntimeError(
        "FD003_FIT_RAW_629C is not a DataFrame."
    )

if not isinstance(
    fit_d_629ab,
    pd.DataFrame,
):
    raise RuntimeError(
        "fit_raw_629d is not a DataFrame."
    )

if fit_c_629ab.shape != (19434, 26):
    raise RuntimeError(
        "FD003_FIT_RAW_629C shape changed."
    )

if fit_d_629ab.shape != (19434, 26):
    raise RuntimeError(
        "fit_raw_629d shape changed."
    )


columns_exact_629ab = bool(
    list(
        fit_c_629ab.columns
    )
    ==
    list(
        fit_d_629ab.columns
    )
)

dtypes_exact_629ab = bool(
    [
        str(x)
        for x in fit_c_629ab.dtypes
    ]
    ==
    [
        str(x)
        for x in fit_d_629ab.dtypes
    ]
)

index_exact_629ab = bool(
    fit_c_629ab.index.equals(
        fit_d_629ab.index
    )
)

values_exact_629ab = bool(
    fit_c_629ab.equals(
        fit_d_629ab
    )
)

if not columns_exact_629ab:
    raise RuntimeError(
        "FIT raw candidate column schemas differ."
    )

if not dtypes_exact_629ab:
    raise RuntimeError(
        "FIT raw candidate dtypes differ."
    )

if not values_exact_629ab:
    raise RuntimeError(
        "FD003_FIT_RAW_629C and fit_raw_629d are not exact copies. "
        "Authority cannot be assigned silently."
    )


# =============================================================================
# 4) Fingerprint both candidates
# =============================================================================

FD003_FIT_RAW_629C_SHA256_629AB = _dataframe_sha256_629ab(
    fit_c_629ab
)

FIT_RAW_629D_SHA256_629AB = _dataframe_sha256_629ab(
    fit_d_629ab
)

hashes_exact_629ab = bool(
    FD003_FIT_RAW_629C_SHA256_629AB
    ==
    FIT_RAW_629D_SHA256_629AB
)

if not hashes_exact_629ab:
    raise RuntimeError(
        "Exact DataFrame equality passed but fingerprints differ."
    )


# =============================================================================
# 5) Verify authoritative FIT engine membership
# =============================================================================

fit_engine_ids_629ab = np.asarray(
    FD003_FIT_ENGINE_IDS_629C,
    dtype=np.int64,
).reshape(-1)

if fit_engine_ids_629ab.shape != (80,):
    raise RuntimeError(
        "FD003 FIT engine-ID vector is not length 80."
    )

if len(
    np.unique(
        fit_engine_ids_629ab
    )
) != 80:
    raise RuntimeError(
        "FD003 FIT engine-ID vector contains duplicates."
    )

raw_fit_unit_ids_629ab = np.sort(
    np.asarray(
        fit_c_629ab[
            "unit_id"
        ].unique(),
        dtype=np.int64,
    )
)

fit_engine_ids_sorted_629ab = np.sort(
    fit_engine_ids_629ab
)

fit_membership_exact_629ab = bool(
    np.array_equal(
        raw_fit_unit_ids_629ab,
        fit_engine_ids_sorted_629ab,
    )
)

if not fit_membership_exact_629ab:
    raise RuntimeError(
        "FD003 FIT raw unit membership disagrees with "
        "FD003_FIT_ENGINE_IDS_629C."
    )


# =============================================================================
# 6) Verify selected feature contract
# =============================================================================

selected_features_629ab = [
    str(x)
    for x in FD003_SELECTED_RAW_FEATURES_629E
]

if len(selected_features_629ab) != 18:
    raise RuntimeError(
        "FD003 selected raw feature count is not 18."
    )

if len(set(selected_features_629ab)) != 18:
    raise RuntimeError(
        "FD003 selected raw feature list contains duplicates."
    )

missing_selected_features_629ab = [
    feature
    for feature in selected_features_629ab
    if feature not in fit_c_629ab.columns
]

if missing_selected_features_629ab:
    raise RuntimeError(
        "Selected FD003 raw feature(s) missing from FIT raw data: "
        + ", ".join(
            missing_selected_features_629ab
        )
    )


preprocess_contract_629ab = FD003_PREPROCESSING_CONTRACT_629E

if not isinstance(
    preprocess_contract_629ab,
    dict,
):
    raise RuntimeError(
        "FD003 preprocessing contract is not a dict."
    )

contract_selected_629ab = preprocess_contract_629ab.get(
    "selected_raw_features",
    None,
)

if contract_selected_629ab is None:
    raise RuntimeError(
        "FD003 preprocessing contract lacks selected_raw_features."
    )

contract_selected_629ab = [
    str(x)
    for x in contract_selected_629ab
]

feature_contract_exact_629ab = bool(
    selected_features_629ab
    ==
    contract_selected_629ab
)

if not feature_contract_exact_629ab:
    raise RuntimeError(
        "FD003_SELECTED_RAW_FEATURES_629E differs from "
        "the frozen preprocessing contract."
    )


contract_selected_count_629ab = preprocess_contract_629ab.get(
    "selected_feature_count",
    None,
)

if (
    contract_selected_count_629ab is not None
    and
    int(
        contract_selected_count_629ab
    )
    !=
    18
):
    raise RuntimeError(
        "Frozen preprocessing selected_feature_count is not 18."
    )


# =============================================================================
# 7) Verify operating-setting structure without inventing a regime model
# =============================================================================

operating_columns_629ab = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

operating_unique_counts_629ab = {
    column: int(
        fit_c_629ab[
            column
        ].nunique(
            dropna=False
        )
    )
    for column in operating_columns_629ab
}

operating_values_finite_629ab = bool(
    np.isfinite(
        fit_c_629ab[
            operating_columns_629ab
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)

if not operating_values_finite_629ab:
    raise RuntimeError(
        "FD003 FIT operating-setting values contain non-finite values."
    )


# =============================================================================
# 8) Authority lock
# =============================================================================

FD003_AUTHORITATIVE_FIT_RAW_OBJECT_629AB = (
    "FD003_FIT_RAW_629C"
)

FD003_AUTHORITATIVE_FIT_RAW_SHA256_629AB = (
    FD003_FIT_RAW_629C_SHA256_629AB
)

FD003_FIT_RAW_ALIAS_629D_EXACT_629AB = True

FD003_FIT_RAW_AUTHORITY_CONTRACT_629AB = {

    "step":
        "6.29AB",

    "dataset":
        "FD003",

    "authoritative_fit_raw_object":
        "FD003_FIT_RAW_629C",

    "authority_reason":
        "upstream engine-level FIT split object; downstream fit_raw_629d "
        "was independently verified as an exact copy",

    "fit_raw_shape":
        [
            int(
                fit_c_629ab.shape[0]
            ),
            int(
                fit_c_629ab.shape[1]
            ),
        ],

    "fit_raw_sha256":
        FD003_AUTHORITATIVE_FIT_RAW_SHA256_629AB,

    "fit_raw_629d_exact_alias":
        True,

    "fit_engine_count":
        80,

    "fit_engine_membership_exact":
        True,

    "selected_raw_features":
        list(
            selected_features_629ab
        ),

    "selected_feature_count":
        18,

    "selected_feature_contract_exact":
        True,

    "operating_setting_unique_counts":
        dict(
            operating_unique_counts_629ab
        ),

    "corruption_reference_created":
        False,

    "corruption_manifest_created":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

authority_payload_629ab = json.dumps(
    FD003_FIT_RAW_AUTHORITY_CONTRACT_629AB,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_FIT_RAW_AUTHORITY_CONTRACT_SHA256_629AB = hashlib.sha256(
    authority_payload_629ab
).hexdigest()


FD003_CORRUPTION_REFERENCE_CREATED_629AB = False
FD003_CORRUPTION_MANIFEST_CREATED_629AB = False
FD003_CORRUPTION_EXECUTED_629AB = False
FD003_DEV_SCORING_PERFORMED_629AB = False
FD003_THRESHOLD_SELECTED_629AB = False
FD003_POLICY_SELECTED_629AB = False
FD003_TEST_CONTENTS_READ_629AB = False
FD003_RUL_CONTENTS_READ_629AB = False

all_checks_passed_629ab = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 140)

print(
    "STEP 6.29AB — FD003 AUTHORITATIVE FIT-RAW "
    "IDENTITY + FEATURE-CONTRACT LOCK"
)

print("=" * 140)

print("Corruption reference created   :", False)
print("Corruption manifest created    :", False)
print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FIT-RAW IDENTITY AUDIT ---")

print(
    "FD003_FIT_RAW_629C shape       :",
    fit_c_629ab.shape,
)

print(
    "fit_raw_629d shape             :",
    fit_d_629ab.shape,
)

print(
    "Columns exact                  :",
    columns_exact_629ab,
)

print(
    "Dtypes exact                   :",
    dtypes_exact_629ab,
)

print(
    "Index exact                    :",
    index_exact_629ab,
)

print(
    "Values exact                   :",
    values_exact_629ab,
)

print(
    "Fingerprints exact             :",
    hashes_exact_629ab,
)

print(
    "FIT raw SHA-256                :",
    FD003_AUTHORITATIVE_FIT_RAW_SHA256_629AB,
)

print("\n--- AUTHORITATIVE FIT SPLIT ---")

print(
    "Authoritative object           :",
    FD003_AUTHORITATIVE_FIT_RAW_OBJECT_629AB,
)

print(
    "FIT engines                    :",
    len(
        fit_engine_ids_629ab
    ),
)

print(
    "Raw FIT membership exact       :",
    fit_membership_exact_629ab,
)

print("\n--- FROZEN FEATURE CONTRACT ---")

print(
    "Selected raw feature count     :",
    len(
        selected_features_629ab
    ),
)

print(
    "Selected raw features          :",
    selected_features_629ab,
)

print(
    "Preprocessing contract exact   :",
    feature_contract_exact_629ab,
)

print("\n--- OPERATING-SETTING STRUCTURE ---")

print(
    "Unique value counts            :",
    operating_unique_counts_629ab,
)

print(
    "\nAuthority-contract SHA-256     :",
    FD003_FIT_RAW_AUTHORITY_CONTRACT_SHA256_629AB,
)

print(
    "\nPASS — FD003_FIT_RAW_629C is now the authoritative FIT-raw "
    "source for corruption calibration. fit_raw_629d is verified as "
    "an exact downstream alias."
)

print(
    "Next: construct and freeze FD003 FIT-only corruption magnitude "
    "and dropout-imputation references from this authoritative source, "
    "then build the 315-scenario FD003 manifest. No corruption will be "
    "executed in that reference-construction step."
)

In [ ]:
# STEP 6.29AC
# FD003 HISTORICAL FIT-ONLY CORRUPTION-REFERENCE CONSTRUCTION AUDIT
#
# Purpose:
# Recover the exact historical methodology used to construct:
#   1) corruption magnitude / scale references
#   2) dropout imputation references
#
# NO FD003 corruption reference created.
# NO corruption manifest created.
# NO corruption executed.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ab", False)):
    raise RuntimeError(
        "STEP 6.29AB must pass before STEP 6.29AC."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AB", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AC."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AB", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AC."
    )

required_629ac = [
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC",
    "FD002_CORRUPTION_MANIFEST_CONTRACT_628GD",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
]

missing_629ac = [
    name
    for name in required_629ac
    if name not in globals()
]

if missing_629ac:
    raise RuntimeError(
        "Missing required historical reference object(s): "
        + ", ".join(missing_629ac)
    )


# =============================================================================
# 2) Exact historical objects
# =============================================================================

contracts_629ac = {
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB":
        FD002_CORRUPTION_TRANSFER_CONTRACT_628GB,

    "FD002_CORRUPTION_OPERATOR_CONTRACT_628GC":
        FD002_CORRUPTION_OPERATOR_CONTRACT_628GC,

    "FD002_CORRUPTION_MANIFEST_CONTRACT_628GD":
        FD002_CORRUPTION_MANIFEST_CONTRACT_628GD,
}

scale_ref_629ac = (
    FD002_CORRUPTION_SCALE_REFERENCE_628GB
)

dropout_ref_629ac = (
    FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
)

if not isinstance(scale_ref_629ac, pd.DataFrame):
    raise RuntimeError(
        "Historical corruption scale reference is not a DataFrame."
    )

if not isinstance(dropout_ref_629ac, pd.DataFrame):
    raise RuntimeError(
        "Historical dropout reference is not a DataFrame."
    )


# =============================================================================
# 3) Recursively extract methodology-relevant fields
# =============================================================================

methodology_tokens_629ac = (
    "scale",
    "std",
    "ddof",
    "variance",
    "pooled",
    "within",
    "regime",
    "condition",
    "mean",
    "imput",
    "reference",
    "fit",
    "sensor",
)


def _safe_scalar_repr_629ac(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            bool,
            int,
            float,
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item() if isinstance(value, np.generic) else value

    return f"<{type(value).__name__}>"


def _walk_contract_629ac(
    value,
    path="root",
    rows=None,
    callables=None,
):

    if rows is None:
        rows = []

    if callables is None:
        callables = []

    if callable(value):

        callables.append(
            (
                path,
                value,
            )
        )

        rows.append(
            {
                "path": path,
                "type": "callable",
                "value": getattr(
                    value,
                    "__qualname__",
                    getattr(
                        value,
                        "__name__",
                        type(value).__name__,
                    ),
                ),
            }
        )

        return rows, callables

    if isinstance(value, dict):

        for key, item in value.items():

            child_path = (
                f"{path}.{key}"
            )

            lower_path = (
                child_path.lower()
            )

            if any(
                token in lower_path
                for token in methodology_tokens_629ac
            ):
                rows.append(
                    {
                        "path": child_path,
                        "type": type(item).__name__,
                        "value": _safe_scalar_repr_629ac(
                            item
                        ),
                    }
                )

            _walk_contract_629ac(
                item,
                path=child_path,
                rows=rows,
                callables=callables,
            )

        return rows, callables

    if isinstance(value, (list, tuple)):

        for idx, item in enumerate(value):

            _walk_contract_629ac(
                item,
                path=f"{path}[{idx}]",
                rows=rows,
                callables=callables,
            )

        return rows, callables

    return rows, callables


methodology_rows_629ac = []
embedded_callables_629ac = []

for contract_name_629ac, contract_629ac in contracts_629ac.items():

    rows_local_629ac = []
    callables_local_629ac = []

    _walk_contract_629ac(
        contract_629ac,
        path=contract_name_629ac,
        rows=rows_local_629ac,
        callables=callables_local_629ac,
    )

    methodology_rows_629ac.extend(
        rows_local_629ac
    )

    embedded_callables_629ac.extend(
        callables_local_629ac
    )


FD003_HISTORICAL_REFERENCE_METHOD_FIELDS_629AC = (
    pd.DataFrame(
        methodology_rows_629ac
    )
    .drop_duplicates(
        subset=[
            "path",
            "type",
            "value",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 4) Audit reference-table schemas + attrs only
# =============================================================================

scale_attrs_629ac = dict(
    getattr(
        scale_ref_629ac,
        "attrs",
        {},
    )
)

dropout_attrs_629ac = dict(
    getattr(
        dropout_ref_629ac,
        "attrs",
        {},
    )
)

scale_schema_629ac = {
    "shape":
        tuple(
            int(x)
            for x in scale_ref_629ac.shape
        ),

    "columns":
        [
            str(c)
            for c in scale_ref_629ac.columns
        ],

    "dtypes":
        {
            str(c):
                str(scale_ref_629ac[c].dtype)
            for c in scale_ref_629ac.columns
        },
}

dropout_schema_629ac = {
    "shape":
        tuple(
            int(x)
            for x in dropout_ref_629ac.shape
        ),

    "columns":
        [
            str(c)
            for c in dropout_ref_629ac.columns
        ],

    "dtypes":
        {
            str(c):
                str(dropout_ref_629ac[c].dtype)
            for c in dropout_ref_629ac.columns
        },
}


# =============================================================================
# 5) Recover source of only explicitly embedded callables
# =============================================================================

callable_rows_629ac = []
callable_sources_629ac = {}

seen_callable_ids_629ac = set()

for path_629ac, fn_629ac in embedded_callables_629ac:

    fn_id_629ac = id(
        fn_629ac
    )

    if fn_id_629ac in seen_callable_ids_629ac:
        continue

    seen_callable_ids_629ac.add(
        fn_id_629ac
    )

    try:
        signature_629ac = str(
            inspect.signature(
                fn_629ac
            )
        )
    except Exception:
        signature_629ac = "<unavailable>"

    try:
        source_629ac = inspect.getsource(
            fn_629ac
        )
    except Exception:
        source_629ac = None

    source_sha_629ac = (
        hashlib.sha256(
            source_629ac.encode(
                "utf-8"
            )
        ).hexdigest()
        if source_629ac is not None
        else None
    )

    callable_rows_629ac.append(
        {
            "contract_path":
                path_629ac,

            "callable_name":
                getattr(
                    fn_629ac,
                    "__qualname__",
                    getattr(
                        fn_629ac,
                        "__name__",
                        type(fn_629ac).__name__,
                    ),
                ),

            "signature":
                signature_629ac,

            "source_available":
                source_629ac is not None,

            "source_sha256":
                source_sha_629ac,
        }
    )

    if source_629ac is not None:

        callable_sources_629ac[
            path_629ac
        ] = source_629ac


FD003_HISTORICAL_REFERENCE_CALLABLE_AUDIT_629AC = (
    pd.DataFrame(
        callable_rows_629ac
    )
)


# =============================================================================
# 6) Resolution flags
# =============================================================================

method_text_629ac = " ".join(
    (
        FD003_HISTORICAL_REFERENCE_METHOD_FIELDS_629AC[
            "path"
        ].astype(str).tolist()
        +
        FD003_HISTORICAL_REFERENCE_METHOD_FIELDS_629AC[
            "value"
        ].astype(str).tolist()
    )
).lower()

explicit_ddof_mentioned_629ac = bool(
    "ddof" in method_text_629ac
)

explicit_pooled_rule_mentioned_629ac = bool(
    "pooled" in method_text_629ac
    or
    "within_regime" in method_text_629ac
    or
    "within-regime" in method_text_629ac
)

explicit_imputation_rule_mentioned_629ac = bool(
    "imput" in method_text_629ac
    or
    "mean" in method_text_629ac
)


# =============================================================================
# 7) State flags
# =============================================================================

FD003_CORRUPTION_REFERENCE_CREATED_629AC = False
FD003_CORRUPTION_MANIFEST_CREATED_629AC = False
FD003_CORRUPTION_EXECUTED_629AC = False
FD003_DEV_SCORING_PERFORMED_629AC = False
FD003_THRESHOLD_SELECTED_629AC = False
FD003_POLICY_SELECTED_629AC = False
FD003_TEST_CONTENTS_READ_629AC = False
FD003_RUL_CONTENTS_READ_629AC = False

all_checks_passed_629ac = True


# =============================================================================
# 8) Output
# =============================================================================

print("=" * 144)

print(
    "STEP 6.29AC — FD003 HISTORICAL FIT-ONLY "
    "CORRUPTION-REFERENCE CONSTRUCTION AUDIT"
)

print("=" * 144)

print("FD003 corruption reference created :", False)
print("FD003 corruption manifest created  :", False)
print("Corruption executed                :", False)
print("Model inference performed          :", False)
print("MC-dropout execution performed     :", False)
print("DEV scoring performed              :", False)
print("Threshold selected                 :", False)
print("Policy selected                    :", False)
print("TEST contents read                 :", False)
print("Official RUL contents read         :", False)

print(
    "\n--- METHODOLOGY-RELEVANT HISTORICAL CONTRACT FIELDS ---"
)

if len(
    FD003_HISTORICAL_REFERENCE_METHOD_FIELDS_629AC
) == 0:

    print(
        "<no methodology-relevant contract fields recovered>"
    )

else:

    print(
        FD003_HISTORICAL_REFERENCE_METHOD_FIELDS_629AC.to_string(
            index=False
        )
    )

print(
    "\n--- HISTORICAL SCALE-REFERENCE SCHEMA ---"
)

print(
    scale_schema_629ac
)

print(
    "DataFrame attrs :",
    scale_attrs_629ac,
)

print(
    "\n--- HISTORICAL DROPOUT-IMPUTATION SCHEMA ---"
)

print(
    dropout_schema_629ac
)

print(
    "DataFrame attrs :",
    dropout_attrs_629ac,
)

print(
    "\n--- CONTRACT-EMBEDDED CALLABLE AUDIT ---"
)

if len(
    FD003_HISTORICAL_REFERENCE_CALLABLE_AUDIT_629AC
) == 0:

    print(
        "<no callable embedded in the exact historical contracts>"
    )

else:

    print(
        FD003_HISTORICAL_REFERENCE_CALLABLE_AUDIT_629AC.to_string(
            index=False
        )
    )

    for path_629ac, source_629ac in callable_sources_629ac.items():

        print(
            "\n"
            + "-" * 118
        )

        print(
            path_629ac
        )

        print(
            "-" * 118
        )

        print(
            source_629ac
        )

print(
    "\n--- RESOLUTION INDICATORS ---"
)

print(
    "Explicit ddof information recovered      :",
    explicit_ddof_mentioned_629ac,
)

print(
    "Explicit pooled/within-regime rule found :",
    explicit_pooled_rule_mentioned_629ac,
)

print(
    "Explicit mean/imputation rule found      :",
    explicit_imputation_rule_mentioned_629ac,
)

print(
    "\nPASS — historical reference-construction metadata was audited "
    "without constructing any FD003 corruption reference."
)

print(
    "Next action depends on whether this output resolves the exact "
    "historical scale and imputation conventions. No convention will "
    "be silently guessed."
)

In [ ]:
# STEP 6.29AD
# FD003 HISTORICAL CORRUPTION-REFERENCE FORMULA RESOLUTION
#
# Forensic reconstruction only.
# Recomputes candidate FD002 FIT-only scale/imputation formulas and compares them
# against the already-frozen historical reference tables.
#
# NO FD003 corruption reference created.
# NO corruption manifest created/frozen.
# NO corruption executed.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ac", False)):
    raise RuntimeError(
        "STEP 6.29AC must pass before STEP 6.29AD."
    )

required_629ad = [
    "FD002_FIT_CORRUPTION_REFERENCE_628GB",
    "FD002_CORRUPTION_SCALE_REFERENCE_628GB",
    "FD002_DROPOUT_IMPUTATION_REFERENCE_628GB",
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD003_SELECTED_RAW_FEATURES_629E",
]

missing_629ad = [
    name
    for name in required_629ad
    if name not in globals()
]

if missing_629ad:
    raise RuntimeError(
        "Missing required STEP 6.29AD object(s): "
        + ", ".join(missing_629ad)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AC", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AD."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AC", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AD."
    )


# =============================================================================
# 2) Strict historical FIT-only whitelist
# =============================================================================

hist_fit_629ad = FD002_FIT_CORRUPTION_REFERENCE_628GB
hist_scale_629ad = FD002_CORRUPTION_SCALE_REFERENCE_628GB
hist_dropout_629ad = FD002_DROPOUT_IMPUTATION_REFERENCE_628GB
hist_transfer_629ad = FD002_CORRUPTION_TRANSFER_CONTRACT_628GB

if not isinstance(hist_fit_629ad, pd.DataFrame):
    raise RuntimeError(
        "Historical FD002 FIT corruption reference is not a DataFrame."
    )

if not isinstance(hist_scale_629ad, pd.DataFrame):
    raise RuntimeError(
        "Historical FD002 scale reference is not a DataFrame."
    )

if not isinstance(hist_dropout_629ad, pd.DataFrame):
    raise RuntimeError(
        "Historical FD002 dropout reference is not a DataFrame."
    )

hist_sensors_629ad = [
    str(x)
    for x in hist_transfer_629ad["sensor_order"]
]

if len(hist_sensors_629ad) != 15:
    raise RuntimeError(
        "Historical corruption sensor count is not 15."
    )

required_hist_columns_629ad = (
    ["unit_id", "cycle", "regime_id"]
    + hist_sensors_629ad
)

missing_hist_columns_629ad = [
    c
    for c in required_hist_columns_629ad
    if c not in hist_fit_629ad.columns
]

if missing_hist_columns_629ad:
    raise RuntimeError(
        "Historical FIT reference lacks required whitelisted column(s): "
        + ", ".join(missing_hist_columns_629ad)
    )

# Explicitly use sensor/regime columns only.
# Any resident RUL/target columns are not selected.
hist_fit_view_629ad = hist_fit_629ad[
    required_hist_columns_629ad
].copy()

if not np.isfinite(
    hist_fit_view_629ad[
        hist_sensors_629ad
    ].to_numpy(dtype=np.float64)
).all():
    raise RuntimeError(
        "Historical FIT sensor view contains non-finite values."
    )


# =============================================================================
# 3) Reconstruct candidate historical scale formulas
# =============================================================================

candidate_rows_629ad = []
per_sensor_rows_629ad = []

stored_scale_map_629ad = {
    str(row.sensor): float(row.pooled_within_regime_std)
    for row in hist_scale_629ad.itertuples(index=False)
}

stored_global_map_629ad = {
    str(row.sensor): float(row.global_raw_std)
    for row in hist_scale_629ad.itertuples(index=False)
}

if set(stored_scale_map_629ad) != set(hist_sensors_629ad):
    raise RuntimeError(
        "Historical scale-reference sensor set mismatch."
    )

regimes_629ad = sorted(
    int(x)
    for x in hist_fit_view_629ad["regime_id"].unique()
)

for sensor_629ad in hist_sensors_629ad:

    x_all_629ad = hist_fit_view_629ad[
        sensor_629ad
    ].to_numpy(dtype=np.float64)

    group_arrays_629ad = [
        group_629ad[
            sensor_629ad
        ].to_numpy(dtype=np.float64)
        for _, group_629ad in hist_fit_view_629ad.groupby(
            "regime_id",
            sort=True,
        )
    ]

    counts_629ad = np.asarray(
        [
            len(x)
            for x in group_arrays_629ad
        ],
        dtype=np.float64,
    )

    means_629ad = np.asarray(
        [
            np.mean(x, dtype=np.float64)
            for x in group_arrays_629ad
        ],
        dtype=np.float64,
    )

    sse_629ad = np.asarray(
        [
            np.sum(
                (x - mean) ** 2,
                dtype=np.float64,
            )
            for x, mean in zip(
                group_arrays_629ad,
                means_629ad,
            )
        ],
        dtype=np.float64,
    )

    std0_629ad = np.asarray(
        [
            np.std(
                x,
                ddof=0,
                dtype=np.float64,
            )
            for x in group_arrays_629ad
        ],
        dtype=np.float64,
    )

    std1_629ad = np.asarray(
        [
            np.std(
                x,
                ddof=1,
                dtype=np.float64,
            )
            for x in group_arrays_629ad
        ],
        dtype=np.float64,
    )

    total_n_629ad = float(
        counts_629ad.sum()
    )

    total_df_629ad = float(
        np.sum(
            counts_629ad - 1.0
        )
    )

    formulas_629ad = {
        "population_SSE_over_N":
            float(
                np.sqrt(
                    sse_629ad.sum()
                    /
                    total_n_629ad
                )
            ),

        "sample_pooled_SSE_over_sum_n_minus_1":
            float(
                np.sqrt(
                    sse_629ad.sum()
                    /
                    total_df_629ad
                )
            ),

        "weighted_RMS_group_sample_std_by_n":
            float(
                np.sqrt(
                    np.sum(
                        counts_629ad
                        *
                        (std1_629ad ** 2)
                    )
                    /
                    total_n_629ad
                )
            ),

        "weighted_mean_group_population_std_by_n":
            float(
                np.sum(
                    counts_629ad
                    *
                    std0_629ad
                )
                /
                total_n_629ad
            ),

        "weighted_mean_group_sample_std_by_n":
            float(
                np.sum(
                    counts_629ad
                    *
                    std1_629ad
                )
                /
                total_n_629ad
            ),

        "unweighted_RMS_group_sample_std":
            float(
                np.sqrt(
                    np.mean(
                        std1_629ad ** 2
                    )
                )
            ),
    }

    stored_scale_629ad = stored_scale_map_629ad[
        sensor_629ad
    ]

    stored_global_629ad = stored_global_map_629ad[
        sensor_629ad
    ]

    per_sensor_rows_629ad.append(
        {
            "sensor":
                sensor_629ad,

            "stored_pooled":
                stored_scale_629ad,

            "stored_global":
                stored_global_629ad,

            "global_std_ddof0":
                float(
                    np.std(
                        x_all_629ad,
                        ddof=0,
                        dtype=np.float64,
                    )
                ),

            "global_std_ddof1":
                float(
                    np.std(
                        x_all_629ad,
                        ddof=1,
                        dtype=np.float64,
                    )
                ),

            **formulas_629ad,
        }
    )


per_sensor_df_629ad = pd.DataFrame(
    per_sensor_rows_629ad
)

pooled_candidate_names_629ad = [
    "population_SSE_over_N",
    "sample_pooled_SSE_over_sum_n_minus_1",
    "weighted_RMS_group_sample_std_by_n",
    "weighted_mean_group_population_std_by_n",
    "weighted_mean_group_sample_std_by_n",
    "unweighted_RMS_group_sample_std",
]

for candidate_629ad in pooled_candidate_names_629ad:

    stored_629ad = per_sensor_df_629ad[
        "stored_pooled"
    ].to_numpy(dtype=np.float64)

    rebuilt_629ad = per_sensor_df_629ad[
        candidate_629ad
    ].to_numpy(dtype=np.float64)

    abs_diff_629ad = np.abs(
        stored_629ad
        -
        rebuilt_629ad
    )

    rel_diff_629ad = abs_diff_629ad / np.maximum(
        np.abs(stored_629ad),
        np.finfo(np.float64).tiny,
    )

    candidate_rows_629ad.append(
        {
            "candidate":
                candidate_629ad,

            "max_abs_diff":
                float(abs_diff_629ad.max()),

            "max_rel_diff":
                float(rel_diff_629ad.max()),

            "allclose_rtol1e-12_atol1e-12":
                bool(
                    np.allclose(
                        stored_629ad,
                        rebuilt_629ad,
                        rtol=1e-12,
                        atol=1e-12,
                    )
                ),
        }
    )

FD003_HISTORICAL_POOLED_SCALE_CANDIDATE_AUDIT_629AD = pd.DataFrame(
    candidate_rows_629ad
)


# =============================================================================
# 4) Resolve historical global-raw STD ddof
# =============================================================================

stored_global_vec_629ad = per_sensor_df_629ad[
    "stored_global"
].to_numpy(dtype=np.float64)

global0_vec_629ad = per_sensor_df_629ad[
    "global_std_ddof0"
].to_numpy(dtype=np.float64)

global1_vec_629ad = per_sensor_df_629ad[
    "global_std_ddof1"
].to_numpy(dtype=np.float64)

global_ddof0_max_abs_629ad = float(
    np.max(
        np.abs(
            stored_global_vec_629ad
            -
            global0_vec_629ad
        )
    )
)

global_ddof1_max_abs_629ad = float(
    np.max(
        np.abs(
            stored_global_vec_629ad
            -
            global1_vec_629ad
        )
    )
)

global_ddof0_exactish_629ad = bool(
    np.allclose(
        stored_global_vec_629ad,
        global0_vec_629ad,
        rtol=1e-12,
        atol=1e-12,
    )
)

global_ddof1_exactish_629ad = bool(
    np.allclose(
        stored_global_vec_629ad,
        global1_vec_629ad,
        rtol=1e-12,
        atol=1e-12,
    )
)

if (
    global_ddof0_exactish_629ad
    and
    not global_ddof1_exactish_629ad
):
    resolved_global_ddof_629ad = 0

elif (
    global_ddof1_exactish_629ad
    and
    not global_ddof0_exactish_629ad
):
    resolved_global_ddof_629ad = 1

else:
    resolved_global_ddof_629ad = None


# =============================================================================
# 5) Reconstruct historical dropout means
# =============================================================================

rebuilt_dropout_rows_629ad = []

for regime_629ad in regimes_629ad:

    regime_df_629ad = hist_fit_view_629ad[
        hist_fit_view_629ad["regime_id"]
        ==
        regime_629ad
    ]

    for sensor_629ad in hist_sensors_629ad:

        rebuilt_dropout_rows_629ad.append(
            {
                "regime_id":
                    int(regime_629ad),

                "sensor":
                    sensor_629ad,

                "mean":
                    float(
                        regime_df_629ad[
                            sensor_629ad
                        ].mean()
                    ),
            }
        )

rebuilt_dropout_629ad = pd.DataFrame(
    rebuilt_dropout_rows_629ad
).sort_values(
    ["regime_id", "sensor"],
    kind="mergesort",
).reset_index(
    drop=True
)

stored_dropout_sorted_629ad = hist_dropout_629ad[
    ["regime_id", "sensor", "mean"]
].copy().sort_values(
    ["regime_id", "sensor"],
    kind="mergesort",
).reset_index(
    drop=True
)

dropout_keys_exact_629ad = bool(
    stored_dropout_sorted_629ad[
        ["regime_id", "sensor"]
    ].equals(
        rebuilt_dropout_629ad[
            ["regime_id", "sensor"]
        ]
    )
)

dropout_mean_abs_diff_629ad = np.abs(
    stored_dropout_sorted_629ad[
        "mean"
    ].to_numpy(dtype=np.float64)
    -
    rebuilt_dropout_629ad[
        "mean"
    ].to_numpy(dtype=np.float64)
)

dropout_means_exactish_629ad = bool(
    dropout_keys_exact_629ad
    and
    np.allclose(
        stored_dropout_sorted_629ad[
            "mean"
        ].to_numpy(dtype=np.float64),
        rebuilt_dropout_629ad[
            "mean"
        ].to_numpy(dtype=np.float64),
        rtol=1e-12,
        atol=1e-12,
    )
)

dropout_mean_max_abs_diff_629ad = float(
    dropout_mean_abs_diff_629ad.max()
)


# =============================================================================
# 6) Resolve pooled-scale formula
# =============================================================================

matching_pooled_candidates_629ad = (
    FD003_HISTORICAL_POOLED_SCALE_CANDIDATE_AUDIT_629AD.loc[
        FD003_HISTORICAL_POOLED_SCALE_CANDIDATE_AUDIT_629AD[
            "allclose_rtol1e-12_atol1e-12"
        ],
        "candidate",
    ]
    .astype(str)
    .tolist()
)

pooled_formula_resolved_629ad = (
    len(
        matching_pooled_candidates_629ad
    )
    ==
    1
)

if pooled_formula_resolved_629ad:
    resolved_pooled_formula_629ad = (
        matching_pooled_candidates_629ad[0]
    )
else:
    resolved_pooled_formula_629ad = None


# =============================================================================
# 7) FD003 sensor-scope audit — NO manifest freeze
# =============================================================================

fd003_selected_raw_629ad = [
    str(x)
    for x in FD003_SELECTED_RAW_FEATURES_629E
]

fd003_sensor_candidates_629ad = [
    x
    for x in fd003_selected_raw_629ad
    if x.startswith("sensor_")
]

historical_sensor_set_629ad = set(
    hist_sensors_629ad
)

fd003_sensor_set_629ad = set(
    fd003_sensor_candidates_629ad
)

fd003_added_vs_fd002_629ad = sorted(
    fd003_sensor_set_629ad
    -
    historical_sensor_set_629ad
)

fd003_missing_vs_fd002_629ad = sorted(
    historical_sensor_set_629ad
    -
    fd003_sensor_set_629ad
)

# Historical structure = 21 scenarios per physical sensor:
# bias 6 + noise 3 + drift 6 + dropout 3 + stuck 3.
implied_fd003_scenario_count_629ad = int(
    21
    *
    len(
        fd003_sensor_candidates_629ad
    )
)


# =============================================================================
# 8) Resolution contract
# =============================================================================

FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_629AD = {
    "step":
        "6.29AD",

    "historical_dataset":
        "FD002",

    "historical_fit_rows":
        int(
            len(
                hist_fit_view_629ad
            )
        ),

    "historical_regimes":
        list(
            regimes_629ad
        ),

    "historical_sensor_count":
        int(
            len(
                hist_sensors_629ad
            )
        ),

    "matching_pooled_candidates":
        list(
            matching_pooled_candidates_629ad
        ),

    "pooled_formula_uniquely_resolved":
        bool(
            pooled_formula_resolved_629ad
        ),

    "resolved_pooled_formula":
        resolved_pooled_formula_629ad,

    "global_ddof0_match":
        global_ddof0_exactish_629ad,

    "global_ddof1_match":
        global_ddof1_exactish_629ad,

    "resolved_global_ddof":
        resolved_global_ddof_629ad,

    "dropout_keys_exact":
        dropout_keys_exact_629ad,

    "dropout_means_match":
        dropout_means_exactish_629ad,

    "fd003_selected_physical_sensor_count":
        int(
            len(
                fd003_sensor_candidates_629ad
            )
        ),

    "fd003_selected_physical_sensors":
        list(
            fd003_sensor_candidates_629ad
        ),

    "fd003_added_vs_fd002_sensor_scope":
        list(
            fd003_added_vs_fd002_629ad
        ),

    "fd003_missing_vs_fd002_sensor_scope":
        list(
            fd003_missing_vs_fd002_629ad
        ),

    "implied_scenario_count_if_same_per_sensor_structure":
        implied_fd003_scenario_count_629ad,

    "fd003_manifest_frozen":
        False,

    "fd003_corruption_reference_created":
        False,

    "fd003_corruption_executed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

resolution_payload_629ad = json.dumps(
    FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_629AD,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_SHA256_629AD = hashlib.sha256(
    resolution_payload_629ad
).hexdigest()


FD003_CORRUPTION_REFERENCE_CREATED_629AD = False
FD003_CORRUPTION_MANIFEST_CREATED_629AD = False
FD003_CORRUPTION_EXECUTED_629AD = False
FD003_DEV_SCORING_PERFORMED_629AD = False
FD003_THRESHOLD_SELECTED_629AD = False
FD003_POLICY_SELECTED_629AD = False
FD003_TEST_CONTENTS_READ_629AD = False
FD003_RUL_CONTENTS_READ_629AD = False

all_checks_passed_629ad = bool(
    pooled_formula_resolved_629ad
    and
    resolved_global_ddof_629ad is not None
    and
    dropout_means_exactish_629ad
)


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 146)

print(
    "STEP 6.29AD — FD003 HISTORICAL CORRUPTION-REFERENCE "
    "FORMULA RESOLUTION"
)

print("=" * 146)

print("FD003 corruption reference created :", False)
print("FD003 corruption manifest created  :", False)
print("Corruption executed                :", False)
print("Model inference performed          :", False)
print("MC-dropout execution performed     :", False)
print("DEV scoring performed              :", False)
print("Threshold selected                 :", False)
print("Policy selected                    :", False)
print("TEST contents read                 :", False)
print("Official RUL contents read         :", False)

print(
    "\n--- HISTORICAL POOLED-SCALE CANDIDATE AUDIT ---"
)

print(
    FD003_HISTORICAL_POOLED_SCALE_CANDIDATE_AUDIT_629AD.to_string(
        index=False
    )
)

print(
    "\nMatching pooled candidate(s)       :",
    matching_pooled_candidates_629ad,
)

print(
    "Unique pooled-formula resolution  :",
    pooled_formula_resolved_629ad,
)

print(
    "Resolved pooled formula           :",
    resolved_pooled_formula_629ad,
)

print(
    "\n--- HISTORICAL GLOBAL-STD AUDIT ---"
)

print(
    "ddof=0 max abs difference         :",
    global_ddof0_max_abs_629ad,
)

print(
    "ddof=1 max abs difference         :",
    global_ddof1_max_abs_629ad,
)

print(
    "ddof=0 matches                    :",
    global_ddof0_exactish_629ad,
)

print(
    "ddof=1 matches                    :",
    global_ddof1_exactish_629ad,
)

print(
    "Resolved global ddof              :",
    resolved_global_ddof_629ad,
)

print(
    "\n--- HISTORICAL DROPOUT-MEAN AUDIT ---"
)

print(
    "Reference key pairs exact         :",
    dropout_keys_exact_629ad,
)

print(
    "Maximum mean abs difference       :",
    dropout_mean_max_abs_diff_629ad,
)

print(
    "FIT regime-specific means match   :",
    dropout_means_exactish_629ad,
)

print(
    "\n--- FD003 PHYSICAL-SENSOR SCOPE AUDIT ---"
)

print(
    "Historical FD002 sensors          :",
    hist_sensors_629ad,
)

print(
    "FD003 selected physical sensors   :",
    fd003_sensor_candidates_629ad,
)

print(
    "FD003 physical sensor count       :",
    len(
        fd003_sensor_candidates_629ad
    ),
)

print(
    "Added vs FD002                    :",
    fd003_added_vs_fd002_629ad,
)

print(
    "Missing vs FD002                  :",
    fd003_missing_vs_fd002_629ad,
)

print(
    "Implied scenarios under same "
    "21-per-sensor structure          :",
    implied_fd003_scenario_count_629ad,
)

print(
    "FD003 manifest frozen             :",
    False,
)

print(
    "\nResolution-contract SHA-256       :",
    FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_SHA256_629AD,
)

if all_checks_passed_629ad:

    print(
        "\nPASS — historical FIT-only scale and dropout-imputation "
        "construction conventions are uniquely resolved."
    )

else:

    print(
        "\nUNRESOLVED — at least one historical reference-construction "
        "convention remains ambiguous. No FD003 reference or manifest "
        "was created."
    )

print(
    "Next: only after reviewing these results will we construct the "
    "FD003 FIT-only corruption references and decide the dataset-specific "
    "sensor scope before manifest freeze."
)

In [ ]:
# STEP 6.29AE
# FD003 HISTORICAL CORRUPTION-MANIFEST ORDER / ID-TEMPLATE RESOLUTION
#
# Read-only forensic step.
# NO FD003 corruption reference created.
# NO FD003 manifest created.
# NO corruption executed.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ad", False)):
    raise RuntimeError(
        "STEP 6.29AD must pass before STEP 6.29AE."
    )

required_629ae = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
    "FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_629AD",
]

missing_629ae = [
    name
    for name in required_629ae
    if name not in globals()
]

if missing_629ae:
    raise RuntimeError(
        "Missing required STEP 6.29AE object(s): "
        + ", ".join(missing_629ae)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AD", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AE."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AD", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AE."
    )


# =============================================================================
# 2) Historical manifest + frozen ordering references
# =============================================================================

manifest_629ae = FD002_CORRUPTION_MANIFEST_628GD.copy()

if not isinstance(manifest_629ae, pd.DataFrame):
    raise RuntimeError(
        "Historical corruption manifest is not a DataFrame."
    )

expected_columns_629ae = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if list(manifest_629ae.columns) != expected_columns_629ae:
    raise RuntimeError(
        "Historical manifest schema changed."
    )

if len(manifest_629ae) != 315:
    raise RuntimeError(
        "Historical FD002 manifest is not 315 rows."
    )

hist_sensor_order_629ae = [
    str(x)
    for x in FD002_CORRUPTION_TRANSFER_CONTRACT_628GB[
        "sensor_order"
    ]
]

hist_family_order_629ae = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

hist_severity_order_629ae = [
    "low",
    "medium",
    "high",
]


# =============================================================================
# 3) Scenario-number audit
# =============================================================================

scenario_numbers_629ae = manifest_629ae[
    "scenario_number"
].to_numpy(dtype=np.int64)

scenario_numbers_contiguous_629ae = bool(
    np.array_equal(
        scenario_numbers_629ae,
        np.arange(
            int(scenario_numbers_629ae.min()),
            int(scenario_numbers_629ae.min())
            + len(manifest_629ae),
            dtype=np.int64,
        ),
    )
)

if not scenario_numbers_contiguous_629ae:
    raise RuntimeError(
        "Historical scenario numbers are not contiguous in row order."
    )

scenario_number_start_629ae = int(
    scenario_numbers_629ae.min()
)

scenario_number_end_629ae = int(
    scenario_numbers_629ae.max()
)


# =============================================================================
# 4) Recover family blocks exactly
# =============================================================================

family_block_rows_629ae = []

for family_629ae in hist_family_order_629ae:

    block_629ae = manifest_629ae[
        manifest_629ae["family"]
        ==
        family_629ae
    ]

    if len(block_629ae) == 0:
        raise RuntimeError(
            f"Historical family block missing: {family_629ae}"
        )

    positions_629ae = block_629ae.index.to_numpy(
        dtype=np.int64
    )

    contiguous_629ae = bool(
        np.array_equal(
            positions_629ae,
            np.arange(
                positions_629ae.min(),
                positions_629ae.max() + 1,
                dtype=np.int64,
            ),
        )
    )

    family_block_rows_629ae.append(
        {
            "family":
                family_629ae,

            "row_count":
                int(len(block_629ae)),

            "first_row_index":
                int(positions_629ae.min()),

            "last_row_index":
                int(positions_629ae.max()),

            "contiguous_block":
                contiguous_629ae,

            "first_scenario_number":
                int(
                    block_629ae[
                        "scenario_number"
                    ].iloc[0]
                ),

            "last_scenario_number":
                int(
                    block_629ae[
                        "scenario_number"
                    ].iloc[-1]
                ),
        }
    )

FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AE = pd.DataFrame(
    family_block_rows_629ae
)

if not bool(
    FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AE[
        "contiguous_block"
    ].all()
):
    raise RuntimeError(
        "At least one historical corruption family is not a contiguous block."
    )


# =============================================================================
# 5) Recover exact within-family ordering
# =============================================================================

def _direction_token_629ae(value):

    if pd.isna(value):
        return None

    return int(
        float(value)
    )


def _float_or_none_629ae(value):

    if pd.isna(value):
        return None

    return float(value)


def _int_or_none_629ae(value):

    if pd.isna(value):
        return None

    return int(
        float(value)
    )


ordering_rows_629ae = []

for family_629ae in hist_family_order_629ae:

    block_629ae = (
        manifest_629ae[
            manifest_629ae["family"]
            ==
            family_629ae
        ]
        .reset_index(
            drop=False
        )
    )

    sensor_sequence_629ae = block_629ae[
        "sensor"
    ].astype(str).tolist()

    severity_sequence_629ae = block_629ae[
        "severity"
    ].astype(str).tolist()

    direction_sequence_629ae = [
        _direction_token_629ae(x)
        for x in block_629ae[
            "direction"
        ].tolist()
    ]

    # Candidate A:
    # sensor outer -> severity -> direction (when applicable)
    candidate_a_629ae = []

    for sensor_629ae in hist_sensor_order_629ae:

        for severity_629ae in hist_severity_order_629ae:

            if family_629ae in (
                "bias",
                "drift",
            ):

                for direction_629ae in (
                    -1,
                    1,
                ):

                    candidate_a_629ae.append(
                        (
                            sensor_629ae,
                            severity_629ae,
                            direction_629ae,
                        )
                    )

            else:

                candidate_a_629ae.append(
                    (
                        sensor_629ae,
                        severity_629ae,
                        None,
                    )
                )

    # Candidate B:
    # severity outer -> sensor -> direction (when applicable)
    candidate_b_629ae = []

    for severity_629ae in hist_severity_order_629ae:

        for sensor_629ae in hist_sensor_order_629ae:

            if family_629ae in (
                "bias",
                "drift",
            ):

                for direction_629ae in (
                    -1,
                    1,
                ):

                    candidate_b_629ae.append(
                        (
                            sensor_629ae,
                            severity_629ae,
                            direction_629ae,
                        )
                    )

            else:

                candidate_b_629ae.append(
                    (
                        sensor_629ae,
                        severity_629ae,
                        None,
                    )
                )

    observed_629ae = list(
        zip(
            sensor_sequence_629ae,
            severity_sequence_629ae,
            direction_sequence_629ae,
        )
    )

    candidate_a_exact_629ae = bool(
        observed_629ae
        ==
        candidate_a_629ae
    )

    candidate_b_exact_629ae = bool(
        observed_629ae
        ==
        candidate_b_629ae
    )

    if (
        candidate_a_exact_629ae
        ==
        candidate_b_exact_629ae
    ):
        raise RuntimeError(
            f"Historical ordering for family={family_629ae} "
            "was not uniquely resolved."
        )

    resolved_order_629ae = (
        "sensor_then_severity_then_direction"
        if candidate_a_exact_629ae
        else
        "severity_then_sensor_then_direction"
    )

    ordering_rows_629ae.append(
        {
            "family":
                family_629ae,

            "sensor_then_severity_exact":
                candidate_a_exact_629ae,

            "severity_then_sensor_exact":
                candidate_b_exact_629ae,

            "resolved_order":
                resolved_order_629ae,
        }
    )


FD003_HISTORICAL_WITHIN_FAMILY_ORDER_AUDIT_629AE = pd.DataFrame(
    ordering_rows_629ae
)


# =============================================================================
# 6) Recover exact per-family parameter template
# =============================================================================

template_rows_629ae = []

template_sensor_629ae = hist_sensor_order_629ae[0]

for family_629ae in hist_family_order_629ae:

    family_sensor_rows_629ae = (
        manifest_629ae[
            (
                manifest_629ae["family"]
                ==
                family_629ae
            )
            &
            (
                manifest_629ae["sensor"]
                ==
                template_sensor_629ae
            )
        ]
        .copy()
    )

    for local_index_629ae, row_629ae in enumerate(
        family_sensor_rows_629ae.itertuples(
            index=False
        ),
        start=1,
    ):

        template_rows_629ae.append(
            {
                "family":
                    str(
                        row_629ae.family
                    ),

                "local_index":
                    int(
                        local_index_629ae
                    ),

                "severity":
                    str(
                        row_629ae.severity
                    ),

                "alpha":
                    _float_or_none_629ae(
                        row_629ae.alpha
                    ),

                "direction":
                    _direction_token_629ae(
                        row_629ae.direction
                    ),

                "requested_fraction":
                    _float_or_none_629ae(
                        row_629ae.requested_fraction
                    ),

                "seed":
                    _int_or_none_629ae(
                        row_629ae.seed
                    ),

                "scenario_id_example":
                    str(
                        row_629ae.scenario_id
                    ),
            }
        )


FD003_HISTORICAL_PER_SENSOR_TEMPLATE_629AE = pd.DataFrame(
    template_rows_629ae
)

if len(
    FD003_HISTORICAL_PER_SENSOR_TEMPLATE_629AE
) != 21:
    raise RuntimeError(
        "Historical per-sensor scenario template is not 21 rows."
    )


# =============================================================================
# 7) Scenario-ID normalization audit
# =============================================================================

scenario_id_template_rows_629ae = []

all_ids_sensor_substitutable_629ae = True

for family_629ae in hist_family_order_629ae:

    first_sensor_rows_629ae = (
        manifest_629ae[
            (
                manifest_629ae["family"]
                ==
                family_629ae
            )
            &
            (
                manifest_629ae["sensor"]
                ==
                template_sensor_629ae
            )
        ]
        .copy()
    )

    normalized_reference_ids_629ae = [
        str(x).replace(
            template_sensor_629ae,
            "{sensor}",
        )
        for x in first_sensor_rows_629ae[
            "scenario_id"
        ].tolist()
    ]

    for sensor_629ae in hist_sensor_order_629ae:

        sensor_rows_629ae = (
            manifest_629ae[
                (
                    manifest_629ae["family"]
                    ==
                    family_629ae
                )
                &
                (
                    manifest_629ae["sensor"]
                    ==
                    sensor_629ae
                )
            ]
            .copy()
        )

        normalized_ids_629ae = [
            str(x).replace(
                sensor_629ae,
                "{sensor}",
            )
            for x in sensor_rows_629ae[
                "scenario_id"
            ].tolist()
        ]

        exact_629ae = bool(
            normalized_ids_629ae
            ==
            normalized_reference_ids_629ae
        )

        if not exact_629ae:
            all_ids_sensor_substitutable_629ae = False

        scenario_id_template_rows_629ae.append(
            {
                "family":
                    family_629ae,

                "sensor":
                    sensor_629ae,

                "normalized_id_pattern_exact":
                    exact_629ae,
            }
        )


FD003_HISTORICAL_SCENARIO_ID_AUDIT_629AE = pd.DataFrame(
    scenario_id_template_rows_629ae
)

if not all_ids_sensor_substitutable_629ae:
    raise RuntimeError(
        "Historical scenario IDs are not consistently sensor-substitutable."
    )


FD003_HISTORICAL_SCENARIO_ID_TEMPLATES_629AE = {
    family_629ae: [
        str(x).replace(
            template_sensor_629ae,
            "{sensor}",
        )
        for x in manifest_629ae[
            (
                manifest_629ae["family"]
                ==
                family_629ae
            )
            &
            (
                manifest_629ae["sensor"]
                ==
                template_sensor_629ae
            )
        ]["scenario_id"].tolist()
    ]
    for family_629ae in hist_family_order_629ae
}


# =============================================================================
# 8) Freeze recovery contract only
# =============================================================================

FD003_HISTORICAL_MANIFEST_TEMPLATE_CONTRACT_629AE = {

    "step":
        "6.29AE",

    "historical_dataset":
        "FD002",

    "historical_manifest_rows":
        315,

    "historical_sensor_count":
        15,

    "historical_family_order":
        list(
            hist_family_order_629ae
        ),

    "historical_severity_order":
        list(
            hist_severity_order_629ae
        ),

    "historical_sensor_order":
        list(
            hist_sensor_order_629ae
        ),

    "scenario_number_start":
        scenario_number_start_629ae,

    "scenario_number_end":
        scenario_number_end_629ae,

    "scenario_numbers_contiguous":
        scenario_numbers_contiguous_629ae,

    "within_family_order":
        {
            row["family"]:
                row["resolved_order"]
            for row in ordering_rows_629ae
        },

    "scenario_ids_sensor_substitutable":
        all_ids_sensor_substitutable_629ae,

    "per_sensor_template_rows":
        21,

    "fd003_manifest_created":
        False,

    "fd003_corruption_reference_created":
        False,

    "fd003_corruption_executed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


contract_payload_629ae = json.dumps(
    FD003_HISTORICAL_MANIFEST_TEMPLATE_CONTRACT_629AE,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_HISTORICAL_MANIFEST_TEMPLATE_CONTRACT_SHA256_629AE = (
    hashlib.sha256(
        contract_payload_629ae
    ).hexdigest()
)


FD003_CORRUPTION_REFERENCE_CREATED_629AE = False
FD003_CORRUPTION_MANIFEST_CREATED_629AE = False
FD003_CORRUPTION_EXECUTED_629AE = False
FD003_DEV_SCORING_PERFORMED_629AE = False
FD003_THRESHOLD_SELECTED_629AE = False
FD003_POLICY_SELECTED_629AE = False
FD003_TEST_CONTENTS_READ_629AE = False
FD003_RUL_CONTENTS_READ_629AE = False

all_checks_passed_629ae = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 146)

print(
    "STEP 6.29AE — FD003 HISTORICAL CORRUPTION-MANIFEST "
    "ORDER / ID-TEMPLATE RESOLUTION"
)

print("=" * 146)

print("FD003 corruption reference created :", False)
print("FD003 corruption manifest created  :", False)
print("Corruption executed                :", False)
print("Model inference performed          :", False)
print("MC-dropout execution performed     :", False)
print("DEV scoring performed              :", False)
print("Threshold selected                 :", False)
print("Policy selected                    :", False)
print("TEST contents read                 :", False)
print("Official RUL contents read         :", False)

print(
    "\n--- HISTORICAL FAMILY BLOCK AUDIT ---"
)

print(
    FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AE.to_string(
        index=False
    )
)

print(
    "\n--- HISTORICAL WITHIN-FAMILY ORDER AUDIT ---"
)

print(
    FD003_HISTORICAL_WITHIN_FAMILY_ORDER_AUDIT_629AE.to_string(
        index=False
    )
)

print(
    "\n--- HISTORICAL 21-ROW PER-SENSOR TEMPLATE ---"
)

print(
    FD003_HISTORICAL_PER_SENSOR_TEMPLATE_629AE.to_string(
        index=False
    )
)

print(
    "\n--- NORMALIZED SCENARIO-ID TEMPLATES ---"
)

for family_629ae in hist_family_order_629ae:

    print(
        family_629ae,
        ":",
        FD003_HISTORICAL_SCENARIO_ID_TEMPLATES_629AE[
            family_629ae
        ],
    )

print(
    "\nScenario numbers contiguous       :",
    scenario_numbers_contiguous_629ae,
)

print(
    "Scenario number range             :",
    (
        scenario_number_start_629ae,
        scenario_number_end_629ae,
    ),
)

print(
    "Scenario IDs sensor-substitutable :",
    all_ids_sensor_substitutable_629ae,
)

print(
    "Per-sensor template rows          :",
    len(
        FD003_HISTORICAL_PER_SENSOR_TEMPLATE_629AE
    ),
)

print(
    "\nManifest-template contract SHA-256:",
    FD003_HISTORICAL_MANIFEST_TEMPLATE_CONTRACT_SHA256_629AE,
)

print(
    "\nPASS — exact historical family ordering, within-family ordering, "
    "21-row per-sensor parameter template, and scenario-ID substitution "
    "semantics are now resolved."
)

print(
    "Next: construct and freeze the FD003 FIT-only corruption references "
    "and dataset-specific 336-scenario manifest. No corruption execution "
    "will occur in that freeze step."
)

In [ ]:
# STEP 6.29AE-R1
# FD003 HISTORICAL MANIFEST SEMANTIC-TEMPLATE RECOVERY
#
# Corrects the failed STEP 6.29AE assumption that historical scenario IDs
# must be sensor-substitutable.
#
# This step recovers the scientifically relevant manifest semantics:
# family blocks, within-family ordering, per-sensor parameter template,
# and scenario-number convention.
#
# Historical scenario-ID strings are treated as non-scientific metadata.
# NO FD003 manifest is created here.
# NO corruption/reference/inference/MC/scoring/threshold/TEST/RUL.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ad", False)):
    raise RuntimeError(
        "STEP 6.29AD must pass before STEP 6.29AE-R1."
    )

required_629aer1 = [
    "FD002_CORRUPTION_MANIFEST_628GD",
    "FD002_CORRUPTION_TRANSFER_CONTRACT_628GB",
]

missing_629aer1 = [
    name
    for name in required_629aer1
    if name not in globals()
]

if missing_629aer1:
    raise RuntimeError(
        "Missing required historical object(s): "
        + ", ".join(missing_629aer1)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AD", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AE-R1."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AD", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AE-R1."
    )


# =============================================================================
# 2) Historical manifest
# =============================================================================

manifest_629aer1 = FD002_CORRUPTION_MANIFEST_628GD.copy()

expected_columns_629aer1 = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if not isinstance(manifest_629aer1, pd.DataFrame):
    raise RuntimeError(
        "Historical corruption manifest is not a DataFrame."
    )

if list(manifest_629aer1.columns) != expected_columns_629aer1:
    raise RuntimeError(
        "Historical corruption manifest schema changed."
    )

if len(manifest_629aer1) != 315:
    raise RuntimeError(
        "Historical corruption manifest is not 315 rows."
    )

hist_sensor_order_629aer1 = [
    str(x)
    for x in FD002_CORRUPTION_TRANSFER_CONTRACT_628GB[
        "sensor_order"
    ]
]

hist_family_order_629aer1 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

hist_severity_order_629aer1 = [
    "low",
    "medium",
    "high",
]


# =============================================================================
# 3) Scenario-number convention
# =============================================================================

scenario_numbers_629aer1 = manifest_629aer1[
    "scenario_number"
].to_numpy(dtype=np.int64)

scenario_number_start_629aer1 = int(
    scenario_numbers_629aer1.min()
)

scenario_number_end_629aer1 = int(
    scenario_numbers_629aer1.max()
)

scenario_numbers_contiguous_629aer1 = bool(
    np.array_equal(
        scenario_numbers_629aer1,
        np.arange(
            scenario_number_start_629aer1,
            scenario_number_start_629aer1
            + len(manifest_629aer1),
            dtype=np.int64,
        ),
    )
)

if not scenario_numbers_contiguous_629aer1:
    raise RuntimeError(
        "Historical scenario numbers are not contiguous in manifest row order."
    )


# =============================================================================
# 4) Family-block audit
# =============================================================================

family_block_rows_629aer1 = []

for family_629aer1 in hist_family_order_629aer1:

    positions_629aer1 = manifest_629aer1.index[
        manifest_629aer1["family"]
        ==
        family_629aer1
    ].to_numpy(dtype=np.int64)

    if len(positions_629aer1) == 0:
        raise RuntimeError(
            f"Historical family missing: {family_629aer1}"
        )

    contiguous_629aer1 = bool(
        np.array_equal(
            positions_629aer1,
            np.arange(
                int(positions_629aer1.min()),
                int(positions_629aer1.max()) + 1,
                dtype=np.int64,
            ),
        )
    )

    family_block_rows_629aer1.append(
        {
            "family": family_629aer1,
            "row_count": int(len(positions_629aer1)),
            "first_row": int(positions_629aer1.min()),
            "last_row": int(positions_629aer1.max()),
            "contiguous": contiguous_629aer1,
        }
    )

FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AER1 = pd.DataFrame(
    family_block_rows_629aer1
)

if not bool(
    FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AER1[
        "contiguous"
    ].all()
):
    raise RuntimeError(
        "Historical corruption families are not contiguous manifest blocks."
    )


# =============================================================================
# 5) Resolve within-family ordering from semantic tuples
# =============================================================================

def _dir_or_none_629aer1(x):
    if pd.isna(x):
        return None
    return int(float(x))


ordering_rows_629aer1 = []

for family_629aer1 in hist_family_order_629aer1:

    block_629aer1 = manifest_629aer1[
        manifest_629aer1["family"]
        ==
        family_629aer1
    ].reset_index(drop=True)

    observed_629aer1 = [
        (
            str(row.sensor),
            str(row.severity),
            _dir_or_none_629aer1(row.direction),
        )
        for row in block_629aer1.itertuples(index=False)
    ]

    sensor_outer_629aer1 = []

    for sensor_629aer1 in hist_sensor_order_629aer1:
        for severity_629aer1 in hist_severity_order_629aer1:

            if family_629aer1 in ("bias", "drift"):
                for direction_629aer1 in (-1, 1):
                    sensor_outer_629aer1.append(
                        (
                            sensor_629aer1,
                            severity_629aer1,
                            direction_629aer1,
                        )
                    )
            else:
                sensor_outer_629aer1.append(
                    (
                        sensor_629aer1,
                        severity_629aer1,
                        None,
                    )
                )

    severity_outer_629aer1 = []

    for severity_629aer1 in hist_severity_order_629aer1:
        for sensor_629aer1 in hist_sensor_order_629aer1:

            if family_629aer1 in ("bias", "drift"):
                for direction_629aer1 in (-1, 1):
                    severity_outer_629aer1.append(
                        (
                            sensor_629aer1,
                            severity_629aer1,
                            direction_629aer1,
                        )
                    )
            else:
                severity_outer_629aer1.append(
                    (
                        sensor_629aer1,
                        severity_629aer1,
                        None,
                    )
                )

    sensor_outer_exact_629aer1 = bool(
        observed_629aer1
        ==
        sensor_outer_629aer1
    )

    severity_outer_exact_629aer1 = bool(
        observed_629aer1
        ==
        severity_outer_629aer1
    )

    if (
        int(sensor_outer_exact_629aer1)
        +
        int(severity_outer_exact_629aer1)
        !=
        1
    ):
        raise RuntimeError(
            f"Historical ordering not uniquely resolved for {family_629aer1}."
        )

    ordering_rows_629aer1.append(
        {
            "family": family_629aer1,
            "sensor_outer_exact": sensor_outer_exact_629aer1,
            "severity_outer_exact": severity_outer_exact_629aer1,
            "resolved_order": (
                "sensor_then_severity_then_direction"
                if sensor_outer_exact_629aer1
                else
                "severity_then_sensor_then_direction"
            ),
        }
    )


FD003_HISTORICAL_WITHIN_FAMILY_ORDER_AUDIT_629AER1 = pd.DataFrame(
    ordering_rows_629aer1
)


# =============================================================================
# 6) Verify exact per-sensor semantic parameter template
# =============================================================================

def _canon_float_629aer1(x):
    if pd.isna(x):
        return None
    return float(x)


def _canon_int_629aer1(x):
    if pd.isna(x):
        return None
    return int(float(x))


template_sensor_629aer1 = hist_sensor_order_629aer1[0]

template_rows_629aer1 = []

for family_629aer1 in hist_family_order_629aer1:

    family_reference_629aer1 = manifest_629aer1[
        (
            manifest_629aer1["family"]
            ==
            family_629aer1
        )
        &
        (
            manifest_629aer1["sensor"]
            ==
            template_sensor_629aer1
        )
    ].reset_index(drop=True)

    reference_semantics_629aer1 = [
        (
            str(row.family),
            str(row.severity),
            _canon_float_629aer1(row.alpha),
            _canon_int_629aer1(row.direction),
            _canon_float_629aer1(row.requested_fraction),
            _canon_int_629aer1(row.seed),
        )
        for row in family_reference_629aer1.itertuples(index=False)
    ]

    for local_index_629aer1, semantic_tuple_629aer1 in enumerate(
        reference_semantics_629aer1,
        start=1,
    ):
        template_rows_629aer1.append(
            {
                "family": semantic_tuple_629aer1[0],
                "local_index": int(local_index_629aer1),
                "severity": semantic_tuple_629aer1[1],
                "alpha": semantic_tuple_629aer1[2],
                "direction": semantic_tuple_629aer1[3],
                "requested_fraction": semantic_tuple_629aer1[4],
                "seed": semantic_tuple_629aer1[5],
            }
        )

    for sensor_629aer1 in hist_sensor_order_629aer1:

        sensor_rows_629aer1 = manifest_629aer1[
            (
                manifest_629aer1["family"]
                ==
                family_629aer1
            )
            &
            (
                manifest_629aer1["sensor"]
                ==
                sensor_629aer1
            )
        ].reset_index(drop=True)

        sensor_semantics_629aer1 = [
            (
                str(row.family),
                str(row.severity),
                _canon_float_629aer1(row.alpha),
                _canon_int_629aer1(row.direction),
                _canon_float_629aer1(row.requested_fraction),
                _canon_int_629aer1(row.seed),
            )
            for row in sensor_rows_629aer1.itertuples(index=False)
        ]

        if sensor_semantics_629aer1 != reference_semantics_629aer1:
            raise RuntimeError(
                "Historical per-sensor semantic corruption template differs "
                f"for family={family_629aer1}, sensor={sensor_629aer1}."
            )


FD003_HISTORICAL_PER_SENSOR_SEMANTIC_TEMPLATE_629AER1 = pd.DataFrame(
    template_rows_629aer1
)

if len(
    FD003_HISTORICAL_PER_SENSOR_SEMANTIC_TEMPLATE_629AER1
) != 21:
    raise RuntimeError(
        "Recovered historical per-sensor semantic template is not 21 rows."
    )


# =============================================================================
# 7) Historical IDs are recorded only as opaque examples
# =============================================================================

FD003_HISTORICAL_SCENARIO_ID_EXAMPLES_629AER1 = (
    manifest_629aer1[
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .groupby(
        "family",
        sort=False,
    )
    .head(3)
    .reset_index(drop=True)
)


# =============================================================================
# 8) Prospective FD003 identifier convention
# =============================================================================

FD003_PROSPECTIVE_SCENARIO_ID_RULE_629AER1 = (
    "FD003_S{scenario_number:03d}"
)

FD003_PROSPECTIVE_SCENARIO_NUMBER_START_629AER1 = (
    scenario_number_start_629aer1
)


# =============================================================================
# 9) Recovery contract
# =============================================================================

FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_629AER1 = {

    "step":
        "6.29AE-R1",

    "historical_dataset":
        "FD002",

    "historical_manifest_rows":
        315,

    "scenario_number_start":
        scenario_number_start_629aer1,

    "scenario_number_end":
        scenario_number_end_629aer1,

    "scenario_numbers_contiguous":
        scenario_numbers_contiguous_629aer1,

    "family_order":
        list(
            hist_family_order_629aer1
        ),

    "severity_order":
        list(
            hist_severity_order_629aer1
        ),

    "sensor_order":
        list(
            hist_sensor_order_629aer1
        ),

    "within_family_order":
        {
            row["family"]:
                row["resolved_order"]
            for row in ordering_rows_629aer1
        },

    "per_sensor_semantic_template_rows":
        21,

    "per_sensor_semantic_template_exact_across_historical_sensors":
        True,

    "historical_scenario_id_strings":
        "opaque_metadata_not_required_for_semantic_transfer",

    "fd003_prospective_scenario_id_rule":
        FD003_PROSPECTIVE_SCENARIO_ID_RULE_629AER1,

    "fd003_manifest_created":
        False,

    "fd003_corruption_reference_created":
        False,

    "fd003_corruption_executed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


payload_629aer1 = json.dumps(
    FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_629AER1,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1 = (
    hashlib.sha256(
        payload_629aer1
    ).hexdigest()
)


FD003_CORRUPTION_REFERENCE_CREATED_629AER1 = False
FD003_CORRUPTION_MANIFEST_CREATED_629AER1 = False
FD003_CORRUPTION_EXECUTED_629AER1 = False
FD003_DEV_SCORING_PERFORMED_629AER1 = False
FD003_THRESHOLD_SELECTED_629AER1 = False
FD003_POLICY_SELECTED_629AER1 = False
FD003_TEST_CONTENTS_READ_629AER1 = False
FD003_RUL_CONTENTS_READ_629AER1 = False

all_checks_passed_629aer1 = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 148)

print(
    "STEP 6.29AE-R1 — FD003 HISTORICAL MANIFEST "
    "SEMANTIC-TEMPLATE RECOVERY"
)

print("=" * 148)

print("Prior failure cause                : scenario-ID substitution assumption")
print("FD003 corruption reference created :", False)
print("FD003 corruption manifest created  :", False)
print("Corruption executed                :", False)
print("Model inference performed          :", False)
print("MC-dropout execution performed     :", False)
print("DEV scoring performed              :", False)
print("Threshold selected                 :", False)
print("Policy selected                    :", False)
print("TEST contents read                 :", False)
print("Official RUL contents read         :", False)

print(
    "\n--- HISTORICAL FAMILY BLOCKS ---"
)

print(
    FD003_HISTORICAL_FAMILY_BLOCK_AUDIT_629AER1.to_string(
        index=False
    )
)

print(
    "\n--- HISTORICAL WITHIN-FAMILY ORDER ---"
)

print(
    FD003_HISTORICAL_WITHIN_FAMILY_ORDER_AUDIT_629AER1.to_string(
        index=False
    )
)

print(
    "\n--- EXACT 21-ROW PER-SENSOR SEMANTIC TEMPLATE ---"
)

print(
    FD003_HISTORICAL_PER_SENSOR_SEMANTIC_TEMPLATE_629AER1.to_string(
        index=False
    )
)

print(
    "\n--- HISTORICAL SCENARIO-ID EXAMPLES (OPAQUE METADATA) ---"
)

print(
    FD003_HISTORICAL_SCENARIO_ID_EXAMPLES_629AER1.to_string(
        index=False
    )
)

print(
    "\nScenario-number range             :",
    (
        scenario_number_start_629aer1,
        scenario_number_end_629aer1,
    ),
)

print(
    "Scenario numbers contiguous       :",
    scenario_numbers_contiguous_629aer1,
)

print(
    "Prospective FD003 scenario-ID rule:",
    FD003_PROSPECTIVE_SCENARIO_ID_RULE_629AER1,
)

print(
    "\nSemantic-contract SHA-256         :",
    FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1,
)

print(
    "\nPASS — the scientifically relevant historical manifest semantics "
    "were recovered without requiring historical scenario-ID strings "
    "to be sensor-substitutable."
)

print(
    "Next: construct and freeze FD003 FIT-only corruption references "
    "and the 336-scenario manifest using the recovered semantic template. "
    "No corruption execution will occur in that freeze step."
)

In [ ]:
# STEP 6.29AF
# FD003 FIT-ONLY CORRUPTION-REFERENCE + 336-SCENARIO MANIFEST FREEZE
#
# Constructs and freezes:
#   1) FD003 FIT-only corruption magnitude references
#   2) FD003 FIT-only dropout-imputation references
#   3) FD003 dataset-specific 336-scenario corruption manifest
#
# NO corruption is executed.
# NO model inference / MC execution.
# NO DEV outcome/error scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aer1", False)):
    raise RuntimeError(
        "STEP 6.29AE-R1 must pass before STEP 6.29AF."
    )

required_629af = [
    "FD003_FIT_RAW_629C",
    "FD003_AUTHORITATIVE_FIT_RAW_SHA256_629AB",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_629AD",
    "FD003_HISTORICAL_PER_SENSOR_SEMANTIC_TEMPLATE_629AER1",
    "FD003_PROSPECTIVE_SCENARIO_ID_RULE_629AER1",
    "FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1",
]

missing_629af = [
    name
    for name in required_629af
    if name not in globals()
]

if missing_629af:
    raise RuntimeError(
        "Missing required STEP 6.29AF object(s): "
        + ", ".join(missing_629af)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AER1", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AF."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AER1", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AF."
    )

if bool(
    globals().get(
        "FD003_CORRUPTION_MANIFEST_LOCKED_629AF",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AF has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _json_scalar_629af(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        f"Unsupported canonical scalar type: {type(value).__name__}"
    )


def _records_sha256_629af(df, columns):

    records = []

    for row in df[columns].itertuples(
        index=False,
        name=None,
    ):

        records.append(
            [
                _json_scalar_629af(x)
                for x in row
            ]
        )

    payload = json.dumps(
        records,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


# =============================================================================
# 3) Authoritative FD003 FIT raw source + sensor scope
# =============================================================================

fit_raw_629af = FD003_FIT_RAW_629C

if not isinstance(fit_raw_629af, pd.DataFrame):
    raise RuntimeError(
        "FD003 authoritative FIT raw source is not a DataFrame."
    )

if fit_raw_629af.shape != (19434, 26):
    raise RuntimeError(
        "FD003 authoritative FIT raw shape changed."
    )

selected_raw_629af = [
    str(x)
    for x in FD003_SELECTED_RAW_FEATURES_629E
]

physical_sensors_629af = [
    x
    for x in selected_raw_629af
    if x.startswith("sensor_")
]

expected_physical_sensors_629af = [
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21",
]

if physical_sensors_629af != expected_physical_sensors_629af:
    raise RuntimeError(
        "FD003 physical-sensor scope differs from the "
        "resolved frozen feature contract."
    )

if len(physical_sensors_629af) != 16:
    raise RuntimeError(
        "FD003 physical-sensor count is not 16."
    )

missing_sensor_columns_629af = [
    sensor
    for sensor in physical_sensors_629af
    if sensor not in fit_raw_629af.columns
]

if missing_sensor_columns_629af:
    raise RuntimeError(
        "FD003 FIT raw source is missing selected physical sensor(s): "
        + ", ".join(missing_sensor_columns_629af)
    )

fit_sensor_matrix_629af = fit_raw_629af[
    physical_sensors_629af
].to_numpy(
    dtype=np.float64
)

if not np.isfinite(fit_sensor_matrix_629af).all():
    raise RuntimeError(
        "FD003 FIT sensor matrix contains non-finite values."
    )


# =============================================================================
# 4) Verify resolved historical reference formula
# =============================================================================

formula_resolution_629af = (
    FD003_HISTORICAL_REFERENCE_FORMULA_RESOLUTION_629AD
)

if not bool(
    formula_resolution_629af[
        "pooled_formula_uniquely_resolved"
    ]
):
    raise RuntimeError(
        "Historical pooled-scale formula was not uniquely resolved."
    )

if (
    formula_resolution_629af[
        "resolved_pooled_formula"
    ]
    !=
    "sample_pooled_SSE_over_sum_n_minus_1"
):
    raise RuntimeError(
        "Unexpected resolved historical pooled-scale formula."
    )

if int(
    formula_resolution_629af[
        "resolved_global_ddof"
    ]
) != 1:
    raise RuntimeError(
        "Unexpected resolved historical global-STD ddof."
    )

if not bool(
    formula_resolution_629af[
        "dropout_means_match"
    ]
):
    raise RuntimeError(
        "Historical dropout-imputation convention was not resolved."
    )


# =============================================================================
# 5) FD003 operating-condition convention
#
# FD003 has one operating condition.
# Historical within-regime calibration therefore reduces to one regime.
# =============================================================================

FD003_CORRUPTION_REGIME_COUNT_629AF = 1
FD003_CORRUPTION_REGIME_ID_629AF = 0

fit_n_629af = int(
    len(fit_raw_629af)
)

if fit_n_629af <= 1:
    raise RuntimeError(
        "FD003 FIT population is too small."
    )


# =============================================================================
# 6) FD003 FIT-only corruption scale reference
# =============================================================================

scale_rows_629af = []

for sensor_629af in physical_sensors_629af:

    values_629af = fit_raw_629af[
        sensor_629af
    ].to_numpy(
        dtype=np.float64
    )

    mean_629af = float(
        np.mean(
            values_629af,
            dtype=np.float64,
        )
    )

    sse_629af = float(
        np.sum(
            (values_629af - mean_629af) ** 2,
            dtype=np.float64,
        )
    )

    pooled_sample_std_629af = float(
        np.sqrt(
            sse_629af
            /
            float(fit_n_629af - 1)
        )
    )

    global_raw_std_629af = float(
        np.std(
            values_629af,
            ddof=1,
            dtype=np.float64,
        )
    )

    if (
        not np.isfinite(pooled_sample_std_629af)
        or pooled_sample_std_629af <= 0.0
    ):
        raise RuntimeError(
            f"Invalid FD003 pooled scale for {sensor_629af}."
        )

    if not np.isfinite(global_raw_std_629af):
        raise RuntimeError(
            f"Invalid global raw STD for {sensor_629af}."
        )

    if not np.isclose(
        pooled_sample_std_629af,
        global_raw_std_629af,
        rtol=1e-12,
        atol=1e-12,
    ):
        raise RuntimeError(
            "Single-condition pooled scale does not match "
            f"global ddof=1 STD for {sensor_629af}."
        )

    scale_rows_629af.append(
        {
            "sensor":
                sensor_629af,

            "pooled_within_regime_std":
                pooled_sample_std_629af,

            "global_raw_std":
                global_raw_std_629af,

            "global_to_within_ratio":
                float(
                    global_raw_std_629af
                    /
                    pooled_sample_std_629af
                ),
        }
    )


FD003_CORRUPTION_SCALE_REFERENCE_629AF = pd.DataFrame(
    scale_rows_629af,
    columns=[
        "sensor",
        "pooled_within_regime_std",
        "global_raw_std",
        "global_to_within_ratio",
    ],
)

if FD003_CORRUPTION_SCALE_REFERENCE_629AF.shape != (16, 4):
    raise RuntimeError(
        "FD003 corruption scale reference shape is not (16, 4)."
    )

if not np.allclose(
    FD003_CORRUPTION_SCALE_REFERENCE_629AF[
        "global_to_within_ratio"
    ].to_numpy(dtype=np.float64),
    np.ones(16, dtype=np.float64),
    rtol=1e-12,
    atol=1e-12,
):
    raise RuntimeError(
        "FD003 global/within scale ratios are not 1."
    )


# =============================================================================
# 7) FD003 FIT-only dropout-imputation reference
# =============================================================================

dropout_rows_629af = []

for sensor_629af in physical_sensors_629af:

    dropout_mean_629af = float(
        fit_raw_629af[
            sensor_629af
        ].mean()
    )

    if not np.isfinite(dropout_mean_629af):
        raise RuntimeError(
            f"Invalid FD003 dropout mean for {sensor_629af}."
        )

    dropout_rows_629af.append(
        {
            "regime_id": 0,
            "sensor": sensor_629af,
            "mean": dropout_mean_629af,
        }
    )


FD003_DROPOUT_IMPUTATION_REFERENCE_629AF = pd.DataFrame(
    dropout_rows_629af,
    columns=[
        "regime_id",
        "sensor",
        "mean",
    ],
)

if FD003_DROPOUT_IMPUTATION_REFERENCE_629AF.shape != (16, 3):
    raise RuntimeError(
        "FD003 dropout reference shape is not (16, 3)."
    )

if not bool(
    (
        FD003_DROPOUT_IMPUTATION_REFERENCE_629AF[
            "regime_id"
        ]
        ==
        0
    ).all()
):
    raise RuntimeError(
        "FD003 dropout reference does not use regime_id=0."
    )


FD003_SENSOR_SCALE_629AF = {
    str(row.sensor):
        float(row.pooled_within_regime_std)
    for row in FD003_CORRUPTION_SCALE_REFERENCE_629AF.itertuples(
        index=False
    )
}

FD003_DROPOUT_MEAN_629AF = {
    (
        int(row.regime_id),
        str(row.sensor),
    ):
        float(row.mean)
    for row in FD003_DROPOUT_IMPUTATION_REFERENCE_629AF.itertuples(
        index=False
    )
}


# =============================================================================
# 8) Recover frozen 21-row semantic template
# =============================================================================

template_629af = (
    FD003_HISTORICAL_PER_SENSOR_SEMANTIC_TEMPLATE_629AER1
    .copy()
)

expected_template_columns_629af = [
    "family",
    "local_index",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if list(template_629af.columns) != expected_template_columns_629af:
    raise RuntimeError(
        "Historical semantic-template schema changed."
    )

if len(template_629af) != 21:
    raise RuntimeError(
        "Historical semantic template is not 21 rows."
    )

family_order_629af = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

expected_family_template_counts_629af = {
    "bias": 6,
    "noise": 3,
    "drift": 6,
    "dropout": 3,
    "stuck": 3,
}

actual_family_template_counts_629af = (
    template_629af
    .groupby(
        "family",
        sort=False,
    )
    .size()
    .astype(int)
    .to_dict()
)

if (
    actual_family_template_counts_629af
    !=
    expected_family_template_counts_629af
):
    raise RuntimeError(
        "Historical semantic-template family counts changed."
    )


# =============================================================================
# 9) Build FD003 336-scenario manifest
# =============================================================================

scenario_id_rule_629af = (
    FD003_PROSPECTIVE_SCENARIO_ID_RULE_629AER1
)

manifest_rows_629af = []
scenario_number_629af = 1

for family_629af in family_order_629af:

    family_template_629af = (
        template_629af[
            template_629af["family"]
            ==
            family_629af
        ]
        .sort_values(
            "local_index",
            kind="mergesort",
        )
    )

    for sensor_629af in physical_sensors_629af:

        for row_629af in family_template_629af.itertuples(
            index=False
        ):

            alpha_629af = (
                None
                if pd.isna(row_629af.alpha)
                else float(row_629af.alpha)
            )

            direction_629af = (
                None
                if pd.isna(row_629af.direction)
                else int(float(row_629af.direction))
            )

            fraction_629af = (
                None
                if pd.isna(row_629af.requested_fraction)
                else float(row_629af.requested_fraction)
            )

            seed_629af = (
                None
                if pd.isna(row_629af.seed)
                else int(float(row_629af.seed))
            )

            scenario_id_629af = scenario_id_rule_629af.format(
                scenario_number=scenario_number_629af
            )

            manifest_rows_629af.append(
                {
                    "scenario_id":
                        scenario_id_629af,

                    "scenario_number":
                        int(scenario_number_629af),

                    "family":
                        family_629af,

                    "sensor":
                        sensor_629af,

                    "severity":
                        str(row_629af.severity),

                    "alpha":
                        alpha_629af,

                    "direction":
                        direction_629af,

                    "requested_fraction":
                        fraction_629af,

                    "seed":
                        seed_629af,
                }
            )

            scenario_number_629af += 1


FD003_CORRUPTION_MANIFEST_629AF = pd.DataFrame(
    manifest_rows_629af,
    columns=[
        "scenario_id",
        "scenario_number",
        "family",
        "sensor",
        "severity",
        "alpha",
        "direction",
        "requested_fraction",
        "seed",
    ],
)


# =============================================================================
# 10) Manifest audits
# =============================================================================

if FD003_CORRUPTION_MANIFEST_629AF.shape != (336, 9):
    raise RuntimeError(
        "FD003 corruption manifest shape is not (336, 9)."
    )

if not FD003_CORRUPTION_MANIFEST_629AF[
    "scenario_id"
].is_unique:
    raise RuntimeError(
        "FD003 scenario IDs are not unique."
    )

if not FD003_CORRUPTION_MANIFEST_629AF[
    "scenario_number"
].is_unique:
    raise RuntimeError(
        "FD003 scenario numbers are not unique."
    )

if not np.array_equal(
    FD003_CORRUPTION_MANIFEST_629AF[
        "scenario_number"
    ].to_numpy(dtype=np.int64),
    np.arange(
        1,
        337,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "FD003 scenario numbers are not exactly 1..336."
    )

manifest_family_counts_629af = (
    FD003_CORRUPTION_MANIFEST_629AF
    .groupby(
        "family",
        sort=False,
    )
    .size()
    .astype(int)
    .to_dict()
)

expected_manifest_family_counts_629af = {
    "bias": 96,
    "noise": 48,
    "drift": 96,
    "dropout": 48,
    "stuck": 48,
}

if (
    manifest_family_counts_629af
    !=
    expected_manifest_family_counts_629af
):
    raise RuntimeError(
        "FD003 manifest family counts differ from expected."
    )

manifest_sensor_counts_629af = (
    FD003_CORRUPTION_MANIFEST_629AF
    .groupby(
        "sensor",
        sort=False,
    )
    .size()
    .astype(int)
)

if not bool(
    (
        manifest_sensor_counts_629af
        ==
        21
    ).all()
):
    raise RuntimeError(
        "Each FD003 sensor does not have exactly 21 scenarios."
    )

observed_family_order_629af = list(
    dict.fromkeys(
        FD003_CORRUPTION_MANIFEST_629AF[
            "family"
        ].tolist()
    )
)

if observed_family_order_629af != family_order_629af:
    raise RuntimeError(
        "FD003 manifest family-block order changed."
    )


# =============================================================================
# 11) Freeze fingerprints
# =============================================================================

FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF = (
    _records_sha256_629af(
        FD003_CORRUPTION_SCALE_REFERENCE_629AF,
        [
            "sensor",
            "pooled_within_regime_std",
            "global_raw_std",
            "global_to_within_ratio",
        ],
    )
)

FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF = (
    _records_sha256_629af(
        FD003_DROPOUT_IMPUTATION_REFERENCE_629AF,
        [
            "regime_id",
            "sensor",
            "mean",
        ],
    )
)

FD003_CORRUPTION_MANIFEST_SHA256_629AF = (
    _records_sha256_629af(
        FD003_CORRUPTION_MANIFEST_629AF,
        [
            "scenario_id",
            "scenario_number",
            "family",
            "sensor",
            "severity",
            "alpha",
            "direction",
            "requested_fraction",
            "seed",
        ],
    )
)


# =============================================================================
# 12) Freeze protocol contract
# =============================================================================

FD003_CORRUPTION_PROTOCOL_629AF = {

    "step":
        "6.29AF",

    "dataset":
        "FD003",

    "corruption_space":
        "raw sensor measurement space",

    "corruption_stage":
        "before FIT-derived normalization and 30-cycle sequence creation",

    "source_fit_raw_object":
        "FD003_FIT_RAW_629C",

    "source_fit_raw_sha256":
        FD003_AUTHORITATIVE_FIT_RAW_SHA256_629AB,

    "operating_condition_count":
        1,

    "corruption_regime_convention":
        "single FD003 operating condition -> regime_id 0 for all rows",

    "scale_rule":
        "FIT-only sample-pooled within-regime standard deviation: "
        "sqrt(sum within-regime SSE / sum(n_regime - 1))",

    "global_raw_std_ddof":
        1,

    "dropout_imputation_rule":
        "FIT-only regime-specific sensor mean; single FD003 regime_id 0",

    "families":
        list(family_order_629af),

    "physical_sensors":
        list(physical_sensors_629af),

    "physical_sensor_count":
        16,

    "scenario_count":
        336,

    "scenario_count_per_sensor":
        21,

    "family_scenario_counts":
        dict(expected_manifest_family_counts_629af),

    "within_family_order":
        "sensor_then_severity_then_direction",

    "scenario_number_range":
        [1, 336],

    "scenario_id_rule":
        scenario_id_rule_629af,

    "noise_seed":
        42,

    "dropout_seed":
        42,

    "historical_manifest_semantic_contract_sha256":
        FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1,

    "scale_reference_sha256":
        FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF,

    "dropout_reference_sha256":
        FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF,

    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "fd002_numeric_scale_values_reused":
        False,

    "fd002_numeric_dropout_means_reused":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


protocol_payload_629af = json.dumps(
    FD003_CORRUPTION_PROTOCOL_629AF,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

FD003_CORRUPTION_PROTOCOL_SHA256_629AF = hashlib.sha256(
    protocol_payload_629af
).hexdigest()


# =============================================================================
# 13) Lock flags
# =============================================================================

FD003_CORRUPTION_REFERENCE_CREATED_629AF = True
FD003_CORRUPTION_REFERENCE_LOCKED_629AF = True

FD003_CORRUPTION_MANIFEST_CREATED_629AF = True
FD003_CORRUPTION_MANIFEST_LOCKED_629AF = True

FD003_CORRUPTION_PROTOCOL_LOCKED_629AF = True

FD003_CORRUPTION_EXECUTED_629AF = False
FD003_DEV_SCORING_PERFORMED_629AF = False
FD003_THRESHOLD_SELECTED_629AF = False
FD003_POLICY_SELECTED_629AF = False
FD003_TEST_CONTENTS_READ_629AF = False
FD003_RUL_CONTENTS_READ_629AF = False

all_checks_passed_629af = True


# =============================================================================
# 14) Output
# =============================================================================

print("=" * 150)

print(
    "STEP 6.29AF — FD003 FIT-ONLY CORRUPTION-REFERENCE "
    "+ 336-SCENARIO MANIFEST FREEZE"
)

print("=" * 150)

print("Corruption reference created   :", True)
print("Corruption reference locked    :", True)
print("Corruption manifest created    :", True)
print("Corruption manifest locked     :", True)
print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FD003 FIT-ONLY CORRUPTION SCALE REFERENCE ---"
)

print(
    FD003_CORRUPTION_SCALE_REFERENCE_629AF.to_string(
        index=False
    )
)

print(
    "\nScale-reference SHA-256        :",
    FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF,
)

print(
    "\n--- FD003 FIT-ONLY DROPOUT IMPUTATION REFERENCE ---"
)

print(
    FD003_DROPOUT_IMPUTATION_REFERENCE_629AF.to_string(
        index=False
    )
)

print(
    "\nDropout-reference SHA-256      :",
    FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF,
)

print(
    "\n--- FD003 CORRUPTION MANIFEST SUMMARY ---"
)

print(
    "Manifest shape                 :",
    FD003_CORRUPTION_MANIFEST_629AF.shape,
)

print(
    "Physical sensors               :",
    physical_sensors_629af,
)

print(
    "Scenarios per sensor           :",
    sorted(
        manifest_sensor_counts_629af.unique().tolist()
    ),
)

print(
    "Family counts                  :",
    manifest_family_counts_629af,
)

print(
    "Scenario-number range          :",
    (
        int(
            FD003_CORRUPTION_MANIFEST_629AF[
                "scenario_number"
            ].min()
        ),
        int(
            FD003_CORRUPTION_MANIFEST_629AF[
                "scenario_number"
            ].max()
        ),
    ),
)

print("\nFirst 12 manifest rows:")

print(
    FD003_CORRUPTION_MANIFEST_629AF.head(
        12
    ).to_string(
        index=False
    )
)

print("\nLast 12 manifest rows:")

print(
    FD003_CORRUPTION_MANIFEST_629AF.tail(
        12
    ).to_string(
        index=False
    )
)

print(
    "\nManifest SHA-256               :",
    FD003_CORRUPTION_MANIFEST_SHA256_629AF,
)

print(
    "Corruption-protocol SHA-256    :",
    FD003_CORRUPTION_PROTOCOL_SHA256_629AF,
)

print(
    "\nPASS — FD003 FIT-only corruption magnitude/imputation references "
    "and the dataset-specific 336-scenario manifest are now constructed, "
    "fingerprinted, and frozen."
)

print(
    "No corruption has yet been executed. Next: build and smoke-test "
    "the FD003 corruption operators/model bridge against clean DEV "
    "reconstruction before launching the full corruption experiment."
)

In [ ]:
# STEP 6.29AG
# FD003 CORRUPTION / MODEL-BRIDGE SOURCE LOCATOR
#
# Metadata-only locator.
# NO corruption execution.
# NO normalization/window reconstruction.
# NO model inference / MC execution.
# NO DEV outcome/error scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_629af", False)):
    raise RuntimeError(
        "STEP 6.29AF must pass before STEP 6.29AG."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AF", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AG."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AF", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AG."
    )

required_locked_629ag = [
    "FD003_CORRUPTION_PROTOCOL_629AF",
    "FD003_CORRUPTION_PROTOCOL_SHA256_629AF",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_SCALE_REFERENCE_629AF",
    "FD003_DROPOUT_IMPUTATION_REFERENCE_629AF",
]

missing_locked_629ag = [
    name
    for name in required_locked_629ag
    if name not in globals()
]

if missing_locked_629ag:
    raise RuntimeError(
        "Missing frozen STEP 6.29AF object(s): "
        + ", ".join(missing_locked_629ag)
    )

include_tokens_629ag = (
    "dev_raw",
    "raw_dev",
    "fit_mean",
    "fit_std",
    "normaliz",
    "scaler",
    "selected_raw_feature",
    "normalized_feature",
    "x_dev",
    "dev_window",
    "model",
    "gru",
    "629e",
    "629f",
    "629j",
    "629k",
    "629l",
)

forbidden_tokens_629ag = (
    "test",
    "official",
    "rul_fd",
    "abs_error",
    "error",
    "score",
    "threshold",
    "accept",
    "abstain",
    "corrupted",
)

candidate_names_629ag = []

for name_629ag in sorted(globals().keys()):

    lower_629ag = name_629ag.lower()

    if any(
        token_629ag in lower_629ag
        for token_629ag in forbidden_tokens_629ag
    ):
        continue

    if not any(
        token_629ag in lower_629ag
        for token_629ag in include_tokens_629ag
    ):
        continue

    candidate_names_629ag.append(
        name_629ag
    )

rows_629ag = []

for name_629ag in candidate_names_629ag:

    obj_629ag = globals()[name_629ag]

    row_629ag = {
        "name": name_629ag,
        "type": type(obj_629ag).__name__,
        "shape_or_len": None,
        "dtype": None,
        "columns_or_keys": None,
        "callable_signature": None,
    }

    if isinstance(obj_629ag, pd.DataFrame):

        row_629ag["shape_or_len"] = tuple(
            int(x)
            for x in obj_629ag.shape
        )

        row_629ag["columns_or_keys"] = [
            str(c)
            for c in obj_629ag.columns
        ]

    elif isinstance(obj_629ag, np.ndarray):

        row_629ag["shape_or_len"] = tuple(
            int(x)
            for x in obj_629ag.shape
        )

        row_629ag["dtype"] = str(
            obj_629ag.dtype
        )

    elif isinstance(obj_629ag, dict):

        row_629ag["shape_or_len"] = int(
            len(obj_629ag)
        )

        row_629ag["columns_or_keys"] = sorted(
            str(k)
            for k in obj_629ag.keys()
        )

    elif isinstance(obj_629ag, (list, tuple)):

        row_629ag["shape_or_len"] = int(
            len(obj_629ag)
        )

    elif callable(obj_629ag):

        try:
            import inspect

            row_629ag["callable_signature"] = str(
                inspect.signature(obj_629ag)
            )
        except Exception:
            row_629ag["callable_signature"] = "<unavailable>"

    else:

        try:
            row_629ag["shape_or_len"] = int(
                len(obj_629ag)
            )
        except Exception:
            row_629ag["shape_or_len"] = "<scalar_or_uninspected>"

    rows_629ag.append(
        row_629ag
    )

FD003_CORRUPTION_BRIDGE_SOURCE_LOCATOR_629AG = pd.DataFrame(
    rows_629ag
)

FD003_CORRUPTION_EXECUTED_629AG = False
FD003_MODEL_INFERENCE_PERFORMED_629AG = False
FD003_MC_EXECUTION_PERFORMED_629AG = False
FD003_DEV_SCORING_PERFORMED_629AG = False
FD003_THRESHOLD_SELECTED_629AG = False
FD003_POLICY_SELECTED_629AG = False
FD003_TEST_CONTENTS_READ_629AG = False
FD003_RUL_CONTENTS_READ_629AG = False

all_checks_passed_629ag = True

print("=" * 148)

print(
    "STEP 6.29AG — FD003 CORRUPTION / MODEL-BRIDGE "
    "SOURCE LOCATOR"
)

print("=" * 148)

print("Corruption executed            :", False)
print("Normalization rebuilt          :", False)
print("DEV windows rebuilt            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- TARGETED RESIDENT BRIDGE-SOURCE INVENTORY ---"
)

if len(FD003_CORRUPTION_BRIDGE_SOURCE_LOCATOR_629AG) == 0:
    print(
        "<no targeted bridge-source objects found>"
    )
else:
    print(
        FD003_CORRUPTION_BRIDGE_SOURCE_LOCATOR_629AG.to_string(
            index=False
        )
    )

print(
    "\nPASS — metadata-only bridge-source locator complete. "
    "No corruption, reconstruction, inference, MC execution, "
    "DEV scoring, TEST, or official RUL access occurred."
)

print(
    "Next: identify the exact authoritative DEV raw frame, FIT scaler "
    "statistics, clean DEV window tensor, and locked model object; then "
    "build a clean-reconstruction smoke test before any corrupted scenario."
)

In [ ]:
# STEP 6.29AH
# FD003 CORRUPTION / MODEL-BRIDGE AUTHORITY + FINGERPRINT LOCK
#
# NO corruption execution.
# NO normalization/window reconstruction.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd
import torch


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ag", False)):
    raise RuntimeError(
        "STEP 6.29AG must pass before STEP 6.29AH."
    )

required_629ah = [
    "FD003_DEV_RAW_629C",
    "dev_raw_629d",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_FIT_FEATURE_MEANS_629E",
    "FD003_FIT_FEATURE_STDS_629E",
    "FD003_FIT_MEANS_SHA256_629E",
    "FD003_FIT_STDS_SHA256_629E",
    "FD003_PREPROCESSING_CONTRACT_629E",
    "X_DEV_FD003_629F",
    "FD003_DEV_WINDOW_SHA256_629F",
    "FD003_WINDOW_CONTRACT_629F",
    "MODEL_FD003_629J",
    "model_629k",
    "FD003_TRAINED_MODEL_STATE_SHA256_629K",
    "_state_dict_sha256_629k",
]

missing_629ah = [
    name
    for name in required_629ah
    if name not in globals()
]

if missing_629ah:
    raise RuntimeError(
        "Missing required STEP 6.29AH object(s): "
        + ", ".join(missing_629ah)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AG", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AH."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AG", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AH."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629ah(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AH hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _dataframe_sha256_629ah(df):

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            "Expected DataFrame."
        )

    h = hashlib.sha256()

    h.update(
        json.dumps(
            [
                str(c)
                for c in df.columns
            ],
            separators=(",", ":"),
        ).encode("utf-8")
    )

    h.update(
        json.dumps(
            [
                str(x)
                for x in df.dtypes
            ],
            separators=(",", ":"),
        ).encode("utf-8")
    )

    idx = np.asarray(
        df.index.to_numpy()
    )

    if idx.dtype.hasobject:

        h.update(
            json.dumps(
                [
                    str(x)
                    for x in idx
                ],
                separators=(",", ":"),
            ).encode("utf-8")
        )

    else:

        h.update(
            np.ascontiguousarray(
                idx
            ).tobytes(order="C")
        )

    for c in df.columns:

        s = df[c]

        if s.dtype == object:

            h.update(
                json.dumps(
                    [
                        None
                        if pd.isna(x)
                        else str(x)
                        for x in s.tolist()
                    ],
                    separators=(",", ":"),
                ).encode("utf-8")
            )

        else:

            h.update(
                np.ascontiguousarray(
                    s.to_numpy()
                ).tobytes(order="C")
            )

    return h.hexdigest()


# =============================================================================
# 3) DEV raw authority audit
# =============================================================================

dev_raw_c_629ah = FD003_DEV_RAW_629C
dev_raw_d_629ah = dev_raw_629d

if not isinstance(
    dev_raw_c_629ah,
    pd.DataFrame,
) or not isinstance(
    dev_raw_d_629ah,
    pd.DataFrame,
):
    raise RuntimeError(
        "FD003 DEV raw candidates are not DataFrames."
    )

if dev_raw_c_629ah.shape != (5286, 26):
    raise RuntimeError(
        "FD003_DEV_RAW_629C shape changed."
    )

if dev_raw_d_629ah.shape != (5286, 26):
    raise RuntimeError(
        "dev_raw_629d shape changed."
    )

dev_columns_exact_629ah = bool(
    list(dev_raw_c_629ah.columns)
    ==
    list(dev_raw_d_629ah.columns)
)

dev_dtypes_exact_629ah = bool(
    [
        str(x)
        for x in dev_raw_c_629ah.dtypes
    ]
    ==
    [
        str(x)
        for x in dev_raw_d_629ah.dtypes
    ]
)

dev_index_exact_629ah = bool(
    dev_raw_c_629ah.index.equals(
        dev_raw_d_629ah.index
    )
)

dev_values_exact_629ah = bool(
    dev_raw_c_629ah.equals(
        dev_raw_d_629ah
    )
)

if not all(
    [
        dev_columns_exact_629ah,
        dev_dtypes_exact_629ah,
        dev_index_exact_629ah,
        dev_values_exact_629ah,
    ]
):
    raise RuntimeError(
        "FD003_DEV_RAW_629C and dev_raw_629d "
        "are not exact aliases."
    )

FD003_DEV_RAW_SHA256_629AH = _dataframe_sha256_629ah(
    dev_raw_c_629ah
)

if (
    FD003_DEV_RAW_SHA256_629AH
    !=
    _dataframe_sha256_629ah(
        dev_raw_d_629ah
    )
):
    raise RuntimeError(
        "FD003 DEV raw alias fingerprints differ."
    )


# =============================================================================
# 4) FIT normalization authority audit
# =============================================================================

raw_features_629ah = [
    str(x)
    for x in FD003_SELECTED_RAW_FEATURES_629E
]

norm_features_629ah = [
    str(x)
    for x in FD003_NORMALIZED_FEATURE_NAMES_629E
]

means_629ah = np.asarray(
    FD003_FIT_FEATURE_MEANS_629E,
    dtype=np.float64,
).reshape(-1)

stds_629ah = np.asarray(
    FD003_FIT_FEATURE_STDS_629E,
    dtype=np.float64,
).reshape(-1)

if len(raw_features_629ah) != 18:
    raise RuntimeError(
        "FD003 raw feature order is not length 18."
    )

if len(norm_features_629ah) != 18:
    raise RuntimeError(
        "FD003 normalized feature order is not length 18."
    )

if means_629ah.shape != (18,):
    raise RuntimeError(
        "FD003 FIT mean vector is not length 18."
    )

if stds_629ah.shape != (18,):
    raise RuntimeError(
        "FD003 FIT STD vector is not length 18."
    )

if not np.isfinite(means_629ah).all():
    raise RuntimeError(
        "FD003 FIT means contain non-finite values."
    )

if not np.isfinite(stds_629ah).all():
    raise RuntimeError(
        "FD003 FIT STDs contain non-finite values."
    )

if not bool(
    (
        stds_629ah
        >
        0.0
    ).all()
):
    raise RuntimeError(
        "FD003 FIT scaler contains non-positive STD."
    )

if (
    _raw_sha256_629ah(
        means_629ah
    )
    !=
    FD003_FIT_MEANS_SHA256_629E
):
    raise RuntimeError(
        "FD003 FIT mean fingerprint changed."
    )

if (
    _raw_sha256_629ah(
        stds_629ah
    )
    !=
    FD003_FIT_STDS_SHA256_629E
):
    raise RuntimeError(
        "FD003 FIT STD fingerprint changed."
    )

expected_norm_names_629ah = [
    f"{name}_z"
    for name in raw_features_629ah
]

if norm_features_629ah != expected_norm_names_629ah:
    raise RuntimeError(
        "FD003 normalized feature names do not match raw feature order."
    )

preprocess_629ah = FD003_PREPROCESSING_CONTRACT_629E

if [
    str(x)
    for x in preprocess_629ah[
        "selected_raw_features"
    ]
] != raw_features_629ah:
    raise RuntimeError(
        "Preprocessing-contract raw feature order mismatch."
    )

if [
    str(x)
    for x in preprocess_629ah[
        "normalized_feature_names"
    ]
] != norm_features_629ah:
    raise RuntimeError(
        "Preprocessing-contract normalized feature order mismatch."
    )

if int(
    preprocess_629ah[
        "std_ddof"
    ]
) != 0:
    raise RuntimeError(
        "Unexpected FD003 normalization std_ddof."
    )


# =============================================================================
# 5) Frozen clean DEV window authority
# =============================================================================

x_dev_629ah = np.asarray(
    X_DEV_FD003_629F
)

if x_dev_629ah.shape != (4706, 30, 18):
    raise RuntimeError(
        "Frozen clean DEV window tensor shape changed."
    )

if str(
    x_dev_629ah.dtype
) != "float32":
    raise RuntimeError(
        "Frozen clean DEV window dtype changed."
    )

if not np.isfinite(
    x_dev_629ah
).all():
    raise RuntimeError(
        "Frozen clean DEV windows contain non-finite values."
    )

if (
    _raw_sha256_629ah(
        x_dev_629ah
    )
    !=
    FD003_DEV_WINDOW_SHA256_629F
):
    raise RuntimeError(
        "Frozen clean DEV window fingerprint changed."
    )

window_contract_629ah = FD003_WINDOW_CONTRACT_629F

if int(
    window_contract_629ah[
        "window_length"
    ]
) != 30:
    raise RuntimeError(
        "Unexpected FD003 window length."
    )

if int(
    window_contract_629ah[
        "model_feature_count"
    ]
) != 18:
    raise RuntimeError(
        "Unexpected FD003 model feature count."
    )

window_model_features_629ah = [
    str(x)
    for x in window_contract_629ah[
        "model_features"
    ]
]

if window_model_features_629ah != norm_features_629ah:
    raise RuntimeError(
        "Window-contract feature order differs from "
        "the frozen normalized feature order."
    )


# =============================================================================
# 6) Trained-model alias audit
# =============================================================================

model_candidate_names_629ah = [
    "MODEL_FD003_629J",
    "model_629k",
    "model_629m",
    "model_629r",
]

model_rows_629ah = []

for name_629ah in model_candidate_names_629ah:

    if name_629ah not in globals():
        continue

    obj_629ah = globals()[
        name_629ah
    ]

    is_module_629ah = isinstance(
        obj_629ah,
        torch.nn.Module,
    )

    state_sha_629ah = (
        _state_dict_sha256_629k(
            obj_629ah
        )
        if is_module_629ah
        else None
    )

    model_rows_629ah.append(
        {
            "name":
                name_629ah,

            "class_name":
                type(
                    obj_629ah
                ).__name__,

            "object_id":
                int(
                    id(
                        obj_629ah
                    )
                ),

            "is_torch_module":
                bool(
                    is_module_629ah
                ),

            "training_mode":
                (
                    bool(
                        obj_629ah.training
                    )
                    if is_module_629ah
                    else None
                ),

            "state_sha256":
                state_sha_629ah,

            "matches_trained_state":
                bool(
                    state_sha_629ah
                    ==
                    FD003_TRAINED_MODEL_STATE_SHA256_629K
                )
                if is_module_629ah
                else False,
        }
    )


FD003_BRIDGE_MODEL_ALIAS_AUDIT_629AH = pd.DataFrame(
    model_rows_629ah
)

for required_model_name_629ah in (
    "MODEL_FD003_629J",
    "model_629k",
):

    row_629ah = FD003_BRIDGE_MODEL_ALIAS_AUDIT_629AH[
        FD003_BRIDGE_MODEL_ALIAS_AUDIT_629AH[
            "name"
        ]
        ==
        required_model_name_629ah
    ]

    if len(row_629ah) != 1:
        raise RuntimeError(
            f"{required_model_name_629ah} missing from model audit."
        )

    if not bool(
        row_629ah.iloc[0][
            "matches_trained_state"
        ]
    ):
        raise RuntimeError(
            f"{required_model_name_629ah} does not match "
            "the frozen trained FD003 model state."
        )


# =============================================================================
# 7) Authority lock
# =============================================================================

FD003_AUTHORITATIVE_DEV_RAW_OBJECT_629AH = (
    "FD003_DEV_RAW_629C"
)

FD003_AUTHORITATIVE_DEV_RAW_629AH = (
    FD003_DEV_RAW_629C
)

FD003_AUTHORITATIVE_BRIDGE_MODEL_NAME_629AH = (
    "MODEL_FD003_629J"
)

FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH = (
    MODEL_FD003_629J
)

if not isinstance(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH,
    torch.nn.Module,
):
    raise RuntimeError(
        "Authoritative FD003 bridge model is not a torch module."
    )

FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH = (
    _state_dict_sha256_629k(
        FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
    )
)

if (
    FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
    !=
    FD003_TRAINED_MODEL_STATE_SHA256_629K
):
    raise RuntimeError(
        "Authoritative bridge model state does not match STEP 6.29K."
    )


# =============================================================================
# 8) Freeze bridge-authority contract
# =============================================================================

FD003_BRIDGE_AUTHORITY_CONTRACT_629AH = {

    "step":
        "6.29AH",

    "dataset":
        "FD003",

    "authoritative_dev_raw_object":
        FD003_AUTHORITATIVE_DEV_RAW_OBJECT_629AH,

    "dev_raw_shape":
        [5286, 26],

    "dev_raw_sha256":
        FD003_DEV_RAW_SHA256_629AH,

    "dev_raw_629d_exact_alias":
        True,

    "raw_feature_order":
        list(
            raw_features_629ah
        ),

    "normalized_feature_order":
        list(
            norm_features_629ah
        ),

    "fit_mean_sha256":
        FD003_FIT_MEANS_SHA256_629E,

    "fit_std_sha256":
        FD003_FIT_STDS_SHA256_629E,

    "normalization_std_ddof":
        0,

    "clean_dev_window_shape":
        [4706, 30, 18],

    "clean_dev_window_sha256":
        FD003_DEV_WINDOW_SHA256_629F,

    "authoritative_model_name":
        FD003_AUTHORITATIVE_BRIDGE_MODEL_NAME_629AH,

    "authoritative_model_state_sha256":
        FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH,

    "expected_trained_model_state_sha256":
        FD003_TRAINED_MODEL_STATE_SHA256_629K,

    "corruption_executed":
        False,

    "normalization_rebuilt":
        False,

    "dev_windows_rebuilt":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


contract_payload_629ah = json.dumps(
    FD003_BRIDGE_AUTHORITY_CONTRACT_629AH,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_BRIDGE_AUTHORITY_CONTRACT_SHA256_629AH = hashlib.sha256(
    contract_payload_629ah
).hexdigest()


FD003_CORRUPTION_EXECUTED_629AH = False
FD003_MODEL_INFERENCE_PERFORMED_629AH = False
FD003_MC_EXECUTION_PERFORMED_629AH = False
FD003_DEV_SCORING_PERFORMED_629AH = False
FD003_THRESHOLD_SELECTED_629AH = False
FD003_POLICY_SELECTED_629AH = False
FD003_TEST_CONTENTS_READ_629AH = False
FD003_RUL_CONTENTS_READ_629AH = False

all_checks_passed_629ah = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 150)

print(
    "STEP 6.29AH — FD003 CORRUPTION / MODEL-BRIDGE "
    "AUTHORITY + FINGERPRINT LOCK"
)

print("=" * 150)

print("Corruption executed            :", False)
print("Normalization rebuilt          :", False)
print("DEV windows rebuilt            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- DEV RAW AUTHORITY ---")

print(
    "Authoritative DEV raw          :",
    FD003_AUTHORITATIVE_DEV_RAW_OBJECT_629AH,
)

print(
    "DEV raw shape                  :",
    FD003_AUTHORITATIVE_DEV_RAW_629AH.shape,
)

print(
    "Columns exact                  :",
    dev_columns_exact_629ah,
)

print(
    "Dtypes exact                   :",
    dev_dtypes_exact_629ah,
)

print(
    "Index exact                    :",
    dev_index_exact_629ah,
)

print(
    "Values exact                   :",
    dev_values_exact_629ah,
)

print(
    "DEV raw SHA-256                :",
    FD003_DEV_RAW_SHA256_629AH,
)

print("\n--- FIT NORMALIZATION AUTHORITY ---")

print(
    "Raw feature count              :",
    len(
        raw_features_629ah
    ),
)

print(
    "Normalized feature count       :",
    len(
        norm_features_629ah
    ),
)

print(
    "FIT means SHA-256              :",
    _raw_sha256_629ah(
        means_629ah
    ),
)

print(
    "FIT stds SHA-256               :",
    _raw_sha256_629ah(
        stds_629ah
    ),
)

print(
    "Normalization ddof             :",
    preprocess_629ah[
        "std_ddof"
    ],
)

print(
    "Feature order exact            :",
    True,
)

print("\n--- CLEAN DEV WINDOW AUTHORITY ---")

print(
    "Shape                          :",
    x_dev_629ah.shape,
)

print(
    "Dtype                          :",
    x_dev_629ah.dtype,
)

print(
    "Window SHA-256                 :",
    _raw_sha256_629ah(
        x_dev_629ah
    ),
)

print(
    "Window feature order exact     :",
    True,
)

print("\n--- TRAINED MODEL ALIAS AUDIT ---")

print(
    FD003_BRIDGE_MODEL_ALIAS_AUDIT_629AH.to_string(
        index=False
    )
)

print(
    "\nAuthoritative model            :",
    FD003_AUTHORITATIVE_BRIDGE_MODEL_NAME_629AH,
)

print(
    "Trained-state SHA-256          :",
    FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH,
)

print(
    "Expected STEP 6.29K SHA-256    :",
    FD003_TRAINED_MODEL_STATE_SHA256_629K,
)

print(
    "\nBridge-authority contract SHA-256:",
    FD003_BRIDGE_AUTHORITY_CONTRACT_SHA256_629AH,
)

print(
    "\nPASS — the authoritative FD003 DEV raw source, FIT normalization "
    "vectors/order, clean DEV windows, and trained model state are now "
    "fingerprinted and locked for corruption-bridge reconstruction."
)

print(
    "Next: rebuild the clean DEV normalization/windows from the authoritative "
    "raw frame and verify exact agreement with the frozen clean DEV windows "
    "before executing even one corrupted scenario."
)

In [ ]:
# STEP 6.29AI
# FD003 CLEAN DEV NORMALIZATION + WINDOW RECONSTRUCTION SMOKE TEST
#
# Rebuilds the clean DEV model input path from the authoritative raw frame
# using only the frozen FIT preprocessing statistics and frozen window rule.
#
# NO corruption is executed.
# NO model inference / MC execution.
# NO DEV outcome/error scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ah", False)):
    raise RuntimeError(
        "STEP 6.29AH must pass before STEP 6.29AI."
    )

required_629ai = [
    "FD003_AUTHORITATIVE_DEV_RAW_629AH",
    "FD003_DEV_RAW_SHA256_629AH",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_FIT_FEATURE_MEANS_629E",
    "FD003_FIT_FEATURE_STDS_629E",
    "FD003_DEV_NORMALIZED_MATRIX_SHA256_629E",
    "X_DEV_FD003_629F",
    "FD003_DEV_WINDOW_SHA256_629F",
    "DEV_WINDOW_UNIT_IDS_FD003_629F",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F",
    "FD003_WINDOW_LENGTH_629F",
    "FD003_WINDOW_STRIDE_629F",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "_state_dict_sha256_629k",
]

missing_629ai = [
    name
    for name in required_629ai
    if name not in globals()
]

if missing_629ai:
    raise RuntimeError(
        "Missing required STEP 6.29AI object(s): "
        + ", ".join(missing_629ai)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AH", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AI."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AH", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AI."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629ai(value):

    arr_629ai = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr_629ai.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AI hashing."
        )

    return hashlib.sha256(
        arr_629ai.tobytes(order="C")
    ).hexdigest()


# =============================================================================
# 3) Verify bridge authorities unchanged
# =============================================================================

dev_raw_629ai = FD003_AUTHORITATIVE_DEV_RAW_629AH

if not isinstance(dev_raw_629ai, pd.DataFrame):
    raise RuntimeError(
        "Authoritative FD003 DEV raw source is not a DataFrame."
    )

if dev_raw_629ai.shape != (5286, 26):
    raise RuntimeError(
        "Authoritative FD003 DEV raw shape changed."
    )

raw_features_629ai = [
    str(x)
    for x in FD003_SELECTED_RAW_FEATURES_629E
]

normalized_features_629ai = [
    str(x)
    for x in FD003_NORMALIZED_FEATURE_NAMES_629E
]

means_629ai = np.asarray(
    FD003_FIT_FEATURE_MEANS_629E,
    dtype=np.float64,
).reshape(-1)

stds_629ai = np.asarray(
    FD003_FIT_FEATURE_STDS_629E,
    dtype=np.float64,
).reshape(-1)

if len(raw_features_629ai) != 18:
    raise RuntimeError(
        "Frozen raw feature count is not 18."
    )

if len(normalized_features_629ai) != 18:
    raise RuntimeError(
        "Frozen normalized feature count is not 18."
    )

if means_629ai.shape != (18,) or stds_629ai.shape != (18,):
    raise RuntimeError(
        "Frozen FIT normalization vectors are not length 18."
    )

if int(FD003_WINDOW_LENGTH_629F) != 30:
    raise RuntimeError(
        "Frozen window length is not 30."
    )

if int(FD003_WINDOW_STRIDE_629F) != 1:
    raise RuntimeError(
        "Frozen window stride is not 1."
    )

model_sha_before_629ai = _state_dict_sha256_629k(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
)

if (
    model_sha_before_629ai
    !=
    FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
):
    raise RuntimeError(
        "FD003 bridge model state changed before clean reconstruction."
    )


# =============================================================================
# 4) Rebuild clean DEV normalization from raw measurements
# =============================================================================

raw_matrix_629ai = dev_raw_629ai[
    raw_features_629ai
].to_numpy(
    dtype=np.float64
)

if raw_matrix_629ai.shape != (5286, 18):
    raise RuntimeError(
        "FD003 clean DEV raw feature matrix shape is not (5286, 18)."
    )

if not np.isfinite(raw_matrix_629ai).all():
    raise RuntimeError(
        "FD003 clean DEV raw feature matrix contains non-finite values."
    )

FD003_CLEAN_DEV_Z_REBUILT_629AI = (
    (
        raw_matrix_629ai
        -
        means_629ai.reshape(1, -1)
    )
    /
    stds_629ai.reshape(1, -1)
).astype(
    np.float64,
    copy=False,
)

if not np.isfinite(
    FD003_CLEAN_DEV_Z_REBUILT_629AI
).all():
    raise RuntimeError(
        "Rebuilt FD003 clean DEV normalized matrix contains non-finite values."
    )

FD003_CLEAN_DEV_Z_REBUILT_SHA256_629AI = _raw_sha256_629ai(
    FD003_CLEAN_DEV_Z_REBUILT_629AI
)

normalization_hash_exact_629ai = bool(
    FD003_CLEAN_DEV_Z_REBUILT_SHA256_629AI
    ==
    FD003_DEV_NORMALIZED_MATRIX_SHA256_629E
)

if not normalization_hash_exact_629ai:
    raise RuntimeError(
        "Clean DEV normalization reconstruction does not reproduce "
        "the frozen STEP 6.29E normalized-matrix fingerprint."
    )


# =============================================================================
# 5) Build reconstruction frame for window formation
# =============================================================================

rebuilt_frame_629ai = dev_raw_629ai[
    [
        "unit_id",
        "cycle",
    ]
].copy()

for j_629ai, feature_629ai in enumerate(
    normalized_features_629ai
):
    rebuilt_frame_629ai[
        feature_629ai
    ] = FD003_CLEAN_DEV_Z_REBUILT_629AI[
        :,
        j_629ai,
    ]


# =============================================================================
# 6) Rebuild clean DEV windows from frozen window semantics
# =============================================================================

window_length_629ai = 30

window_blocks_629ai = []
window_unit_ids_629ai = []
window_endpoint_cycles_629ai = []

for unit_id_629ai, group_629ai in rebuilt_frame_629ai.groupby(
    "unit_id",
    sort=False,
):

    group_sorted_629ai = group_629ai.sort_values(
        "cycle",
        kind="mergesort",
    )

    feature_values_629ai = group_sorted_629ai[
        normalized_features_629ai
    ].to_numpy(
        dtype=np.float64
    )

    cycles_629ai = group_sorted_629ai[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    )

    n_rows_629ai = int(
        len(group_sorted_629ai)
    )

    if n_rows_629ai < window_length_629ai:
        raise RuntimeError(
            f"DEV engine {unit_id_629ai} has fewer than 30 rows."
        )

    for start_629ai in range(
        0,
        n_rows_629ai - window_length_629ai + 1,
        1,
    ):

        end_exclusive_629ai = (
            start_629ai
            +
            window_length_629ai
        )

        window_blocks_629ai.append(
            feature_values_629ai[
                start_629ai:end_exclusive_629ai,
                :,
            ]
        )

        window_unit_ids_629ai.append(
            int(unit_id_629ai)
        )

        window_endpoint_cycles_629ai.append(
            int(
                cycles_629ai[
                    end_exclusive_629ai - 1
                ]
            )
        )


FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI = np.asarray(
    window_blocks_629ai,
    dtype=np.float32,
)

FD003_CLEAN_DEV_WINDOW_UNIT_IDS_REBUILT_629AI = np.asarray(
    window_unit_ids_629ai,
    dtype=np.int64,
)

FD003_CLEAN_DEV_WINDOW_ENDPOINT_CYCLES_REBUILT_629AI = np.asarray(
    window_endpoint_cycles_629ai,
    dtype=np.int64,
)


# =============================================================================
# 7) Exact reconstruction audits
# =============================================================================

if FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI.shape != (4706, 30, 18):
    raise RuntimeError(
        "Rebuilt clean DEV window tensor shape is not (4706, 30, 18)."
    )

if FD003_CLEAN_DEV_WINDOW_UNIT_IDS_REBUILT_629AI.shape != (4706,):
    raise RuntimeError(
        "Rebuilt DEV window-unit vector shape is not (4706,)."
    )

if FD003_CLEAN_DEV_WINDOW_ENDPOINT_CYCLES_REBUILT_629AI.shape != (4706,):
    raise RuntimeError(
        "Rebuilt DEV endpoint-cycle vector shape is not (4706,)."
    )

membership_exact_629ai = bool(
    np.array_equal(
        FD003_CLEAN_DEV_WINDOW_UNIT_IDS_REBUILT_629AI,
        np.asarray(
            DEV_WINDOW_UNIT_IDS_FD003_629F,
            dtype=np.int64,
        ).reshape(-1),
    )
)

endpoint_exact_629ai = bool(
    np.array_equal(
        FD003_CLEAN_DEV_WINDOW_ENDPOINT_CYCLES_REBUILT_629AI,
        np.asarray(
            DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
            dtype=np.int64,
        ).reshape(-1),
    )
)

if not membership_exact_629ai:
    raise RuntimeError(
        "Rebuilt clean DEV window engine membership differs "
        "from frozen STEP 6.29F."
    )

if not endpoint_exact_629ai:
    raise RuntimeError(
        "Rebuilt clean DEV endpoint cycles differ from frozen STEP 6.29F."
    )

FD003_CLEAN_DEV_WINDOWS_REBUILT_SHA256_629AI = _raw_sha256_629ai(
    FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI
)

window_hash_exact_629ai = bool(
    FD003_CLEAN_DEV_WINDOWS_REBUILT_SHA256_629AI
    ==
    FD003_DEV_WINDOW_SHA256_629F
)

frozen_x_629ai = np.asarray(
    X_DEV_FD003_629F,
    dtype=np.float32,
)

max_abs_window_diff_629ai = float(
    np.max(
        np.abs(
            FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI.astype(
                np.float64
            )
            -
            frozen_x_629ai.astype(
                np.float64
            )
        )
    )
)

array_exact_629ai = bool(
    np.array_equal(
        FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI,
        frozen_x_629ai,
    )
)

if not window_hash_exact_629ai:
    raise RuntimeError(
        "Rebuilt clean DEV windows do not reproduce "
        "the frozen STEP 6.29F window fingerprint."
    )

if not array_exact_629ai:
    raise RuntimeError(
        "Rebuilt clean DEV windows are not elementwise exact."
    )


# =============================================================================
# 8) Make reconstructed evidence read-only
# =============================================================================

FD003_CLEAN_DEV_Z_REBUILT_629AI.setflags(
    write=False
)

FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI.setflags(
    write=False
)

FD003_CLEAN_DEV_WINDOW_UNIT_IDS_REBUILT_629AI.setflags(
    write=False
)

FD003_CLEAN_DEV_WINDOW_ENDPOINT_CYCLES_REBUILT_629AI.setflags(
    write=False
)


# =============================================================================
# 9) Verify model remained untouched
# =============================================================================

model_sha_after_629ai = _state_dict_sha256_629k(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
)

model_unchanged_629ai = bool(
    model_sha_after_629ai
    ==
    model_sha_before_629ai
)

if not model_unchanged_629ai:
    raise RuntimeError(
        "FD003 model state changed during clean reconstruction."
    )


# =============================================================================
# 10) Freeze reconstruction contract
# =============================================================================

FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_629AI = {

    "step":
        "6.29AI",

    "dataset":
        "FD003",

    "source_dev_raw":
        "FD003_DEV_RAW_629C",

    "normalization":
        "(raw_feature - frozen_FIT_mean) / frozen_FIT_std",

    "normalization_dtype":
        "float64",

    "normalization_sha256":
        FD003_CLEAN_DEV_Z_REBUILT_SHA256_629AI,

    "frozen_normalization_sha256":
        FD003_DEV_NORMALIZED_MATRIX_SHA256_629E,

    "normalization_hash_exact":
        True,

    "window_length":
        30,

    "window_stride":
        1,

    "window_grouping":
        "unit_id sort=False; stable cycle ascending within engine",

    "window_output_dtype":
        "float32",

    "window_shape":
        [4706, 30, 18],

    "window_sha256":
        FD003_CLEAN_DEV_WINDOWS_REBUILT_SHA256_629AI,

    "frozen_window_sha256":
        FD003_DEV_WINDOW_SHA256_629F,

    "window_hash_exact":
        True,

    "window_array_exact":
        True,

    "window_membership_exact":
        True,

    "window_endpoint_cycles_exact":
        True,

    "model_state_before":
        model_sha_before_629ai,

    "model_state_after":
        model_sha_after_629ai,

    "model_state_unchanged":
        True,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


contract_payload_629ai = json.dumps(
    FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_629AI,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_SHA256_629AI = hashlib.sha256(
    contract_payload_629ai
).hexdigest()


FD003_CLEAN_BRIDGE_RECONSTRUCTION_LOCKED_629AI = True

FD003_CORRUPTION_EXECUTED_629AI = False
FD003_MODEL_INFERENCE_PERFORMED_629AI = False
FD003_MC_EXECUTION_PERFORMED_629AI = False
FD003_DEV_SCORING_PERFORMED_629AI = False
FD003_THRESHOLD_SELECTED_629AI = False
FD003_POLICY_SELECTED_629AI = False
FD003_TEST_CONTENTS_READ_629AI = False
FD003_RUL_CONTENTS_READ_629AI = False

all_checks_passed_629ai = True


# =============================================================================
# 11) Output
# =============================================================================

print("=" * 150)

print(
    "STEP 6.29AI — FD003 CLEAN DEV NORMALIZATION + "
    "WINDOW RECONSTRUCTION SMOKE TEST"
)

print("=" * 150)

print("Corruption executed            :", False)
print("Clean normalization rebuilt    :", True)
print("Clean DEV windows rebuilt      :", True)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- CLEAN NORMALIZATION RECONSTRUCTION ---"
)

print(
    "Rebuilt normalized shape       :",
    FD003_CLEAN_DEV_Z_REBUILT_629AI.shape,
)

print(
    "Rebuilt normalized SHA-256     :",
    FD003_CLEAN_DEV_Z_REBUILT_SHA256_629AI,
)

print(
    "Frozen STEP 6.29E SHA-256      :",
    FD003_DEV_NORMALIZED_MATRIX_SHA256_629E,
)

print(
    "Normalization fingerprint exact:",
    normalization_hash_exact_629ai,
)

print(
    "\n--- CLEAN WINDOW RECONSTRUCTION ---"
)

print(
    "Rebuilt window shape           :",
    FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI.shape,
)

print(
    "Rebuilt window dtype           :",
    FD003_CLEAN_DEV_WINDOWS_REBUILT_629AI.dtype,
)

print(
    "Rebuilt window SHA-256         :",
    FD003_CLEAN_DEV_WINDOWS_REBUILT_SHA256_629AI,
)

print(
    "Frozen STEP 6.29F SHA-256      :",
    FD003_DEV_WINDOW_SHA256_629F,
)

print(
    "Window fingerprint exact       :",
    window_hash_exact_629ai,
)

print(
    "Window array elementwise exact :",
    array_exact_629ai,
)

print(
    "Maximum absolute window diff   :",
    max_abs_window_diff_629ai,
)

print(
    "Window engine membership exact :",
    membership_exact_629ai,
)

print(
    "Endpoint cycles exact          :",
    endpoint_exact_629ai,
)

print(
    "\n--- MODEL STATE SAFETY ---"
)

print(
    "Model SHA before               :",
    model_sha_before_629ai,
)

print(
    "Model SHA after                :",
    model_sha_after_629ai,
)

print(
    "Model state unchanged          :",
    model_unchanged_629ai,
)

print(
    "\nReconstruction-contract SHA-256:",
    FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_SHA256_629AI,
)

print(
    "\nPASS — the raw→normalization→30-cycle-window bridge reproduces "
    "the frozen FD003 clean DEV model input exactly."
)

print(
    "Next: freeze FD003 corruption-operator implementations using the "
    "already-frozen FD003 scale/imputation references, then run one "
    "strict smoke scenario before the full 336-scenario corruption experiment."
)

In [ ]:
# STEP 6.29AJ
# FD003 CORRUPTION-OPERATOR IMPLEMENTATION FREEZE
#
# Defines the FD003 corruption operators by preserving the recovered historical
# semantics and replacing only dataset-specific numeric references with the
# already-frozen FD003 FIT-only scale/imputation references.
#
# NO corruption is executed.
# NO normalization/window reconstruction.
# NO model inference / MC execution.
# NO DEV scoring.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import inspect
import json
import types
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ai", False)):
    raise RuntimeError(
        "STEP 6.29AI must pass before STEP 6.29AJ."
    )

required_629aj = [
    "FD003_CORRUPTION_SCALE_REFERENCE_629AF",
    "FD003_DROPOUT_IMPUTATION_REFERENCE_629AF",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF",
    "FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_PROTOCOL_SHA256_629AF",
    "FD003_HISTORICAL_CORRUPTION_OPERATOR_SEMANTICS_AUDIT_629ZR2",
    "FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1",
]

missing_629aj = [
    name
    for name in required_629aj
    if name not in globals()
]

if missing_629aj:
    raise RuntimeError(
        "Missing required STEP 6.29AJ object(s): "
        + ", ".join(missing_629aj)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AI", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AJ."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AI", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AJ."
    )

if bool(
    globals().get(
        "FD003_CORRUPTION_OPERATOR_CONTRACT_LOCKED_629AJ",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AJ has already been completed."
    )


# =============================================================================
# 2) Canonical frozen-input fingerprint helper
# =============================================================================

def _json_scalar_629aj(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        f"Unsupported canonical scalar type: {type(value).__name__}"
    )


def _records_sha256_629aj(df, columns):

    records_629aj = []

    for row_629aj in df[columns].itertuples(
        index=False,
        name=None,
    ):
        records_629aj.append(
            [
                _json_scalar_629aj(x)
                for x in row_629aj
            ]
        )

    payload_629aj = json.dumps(
        records_629aj,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(
        payload_629aj
    ).hexdigest()


scale_sha_now_629aj = _records_sha256_629aj(
    FD003_CORRUPTION_SCALE_REFERENCE_629AF,
    [
        "sensor",
        "pooled_within_regime_std",
        "global_raw_std",
        "global_to_within_ratio",
    ],
)

dropout_sha_now_629aj = _records_sha256_629aj(
    FD003_DROPOUT_IMPUTATION_REFERENCE_629AF,
    [
        "regime_id",
        "sensor",
        "mean",
    ],
)

manifest_sha_now_629aj = _records_sha256_629aj(
    FD003_CORRUPTION_MANIFEST_629AF,
    [
        "scenario_id",
        "scenario_number",
        "family",
        "sensor",
        "severity",
        "alpha",
        "direction",
        "requested_fraction",
        "seed",
    ],
)

if (
    scale_sha_now_629aj
    !=
    FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF
):
    raise RuntimeError(
        "Frozen FD003 corruption scale reference changed."
    )

if (
    dropout_sha_now_629aj
    !=
    FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF
):
    raise RuntimeError(
        "Frozen FD003 dropout reference changed."
    )

if (
    manifest_sha_now_629aj
    !=
    FD003_CORRUPTION_MANIFEST_SHA256_629AF
):
    raise RuntimeError(
        "Frozen FD003 corruption manifest changed."
    )


# =============================================================================
# 3) Immutable FD003 operator references
# =============================================================================

_sensor_scale_dict_629aj = {
    str(row.sensor):
        float(row.pooled_within_regime_std)
    for row in FD003_CORRUPTION_SCALE_REFERENCE_629AF.itertuples(
        index=False
    )
}

_dropout_mean_dict_629aj = {
    (
        int(row.regime_id),
        str(row.sensor),
    ):
        float(row.mean)
    for row in FD003_DROPOUT_IMPUTATION_REFERENCE_629AF.itertuples(
        index=False
    )
}

FD003_SENSOR_SCALE_OPERATOR_629AJ = types.MappingProxyType(
    _sensor_scale_dict_629aj
)

FD003_DROPOUT_MEAN_OPERATOR_629AJ = types.MappingProxyType(
    _dropout_mean_dict_629aj
)

FD003_OPERATOR_SENSOR_ORDER_629AJ = tuple(
    FD003_CORRUPTION_SCALE_REFERENCE_629AF[
        "sensor"
    ].astype(str).tolist()
)

if len(FD003_OPERATOR_SENSOR_ORDER_629AJ) != 16:
    raise RuntimeError(
        "FD003 operator sensor scope is not 16 sensors."
    )


# =============================================================================
# 4) Base-frame preparation and validation
# =============================================================================

def prepare_fd003_corruption_base_629aj(raw_df):

    if not isinstance(raw_df, pd.DataFrame):
        raise TypeError(
            "raw_df must be a pandas DataFrame."
        )

    required_cols_629aj = [
        "unit_id",
        "cycle",
        *FD003_OPERATOR_SENSOR_ORDER_629AJ,
    ]

    missing_cols_629aj = [
        col_629aj
        for col_629aj in required_cols_629aj
        if col_629aj not in raw_df.columns
    ]

    if missing_cols_629aj:
        raise ValueError(
            "Missing required corruption-base column(s): "
            + ", ".join(missing_cols_629aj)
        )

    prepared_629aj = (
        raw_df
        .copy()
        .reset_index(drop=True)
    )

    prepared_629aj["regime_id"] = np.zeros(
        len(prepared_629aj),
        dtype=np.int64,
    )

    return prepared_629aj


def validate_fd003_sensor_629aj(df, sensor):

    sensor_629aj = str(sensor)

    if sensor_629aj not in FD003_SENSOR_SCALE_OPERATOR_629AJ:
        raise ValueError(
            "Sensor is outside the frozen FD003 corruption scope: "
            + sensor_629aj
        )

    if sensor_629aj not in df.columns:
        raise ValueError(
            "Sensor column is absent from corruption frame: "
            + sensor_629aj
        )

    if "unit_id" not in df.columns:
        raise ValueError(
            "unit_id is required for FD003 corruption."
        )

    if "cycle" not in df.columns:
        raise ValueError(
            "cycle is required for FD003 corruption."
        )


# =============================================================================
# 5) FD003 corruption operators
# =============================================================================

def apply_fd003_bias_629aj(
    df,
    sensor,
    alpha,
    direction,
):

    validate_fd003_sensor_629aj(
        df,
        sensor,
    )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    scale = FD003_SENSOR_SCALE_OPERATOR_629AJ[
        sensor
    ]

    bias_amount = float(
        direction
        *
        alpha
        *
        scale
    )

    corrupted = df.copy()

    corrupted[sensor] = (
        corrupted[sensor].to_numpy(
            dtype=np.float64
        )
        +
        bias_amount
    )

    return (
        corrupted,
        bias_amount,
    )


def apply_fd003_noise_629aj(
    df,
    sensor,
    alpha,
    seed=42,
):

    validate_fd003_sensor_629aj(
        df,
        sensor,
    )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    scale = FD003_SENSOR_SCALE_OPERATOR_629AJ[
        sensor
    ]

    target_std = float(
        alpha
        *
        scale
    )

    rng = np.random.default_rng(
        int(seed)
    )

    base_noise = rng.normal(
        loc=0.0,
        scale=1.0,
        size=len(df),
    )

    noise = (
        base_noise
        *
        target_std
    )

    corrupted = df.copy()

    corrupted[sensor] = (
        corrupted[sensor].to_numpy(
            dtype=np.float64
        )
        +
        noise
    )

    return (
        corrupted,
        noise,
        target_std,
    )


def apply_fd003_drift_629aj(
    df,
    sensor,
    alpha,
    direction,
):

    validate_fd003_sensor_629aj(
        df,
        sensor,
    )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(alpha)

    if (
        not np.isfinite(alpha)
        or
        alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    final_drift = float(
        direction
        *
        alpha
        *
        FD003_SENSOR_SCALE_OPERATOR_629AJ[
            sensor
        ]
    )

    corrupted = df.copy()

    drift_values = np.zeros(
        len(corrupted),
        dtype=np.float64,
    )

    for _, group in corrupted.groupby(
        "unit_id",
        sort=False,
    ):

        group_sorted = group.sort_values(
            "cycle"
        )

        positions = group_sorted.index.to_numpy(
            dtype=np.int64
        )

        n_rows = len(
            positions
        )

        if n_rows == 1:

            engine_drift = np.array(
                [final_drift],
                dtype=np.float64,
            )

        else:

            engine_drift = np.linspace(
                0.0,
                final_drift,
                num=n_rows,
                endpoint=True,
                dtype=np.float64,
            )

        drift_values[
            positions
        ] = engine_drift

    corrupted[sensor] = (
        corrupted[sensor].to_numpy(
            dtype=np.float64
        )
        +
        drift_values
    )

    return (
        corrupted,
        drift_values,
        final_drift,
    )


def apply_fd003_dropout_629aj(
    df,
    sensor,
    dropout_rate,
    seed=42,
):

    validate_fd003_sensor_629aj(
        df,
        sensor,
    )

    dropout_rate = float(
        dropout_rate
    )

    if not (
        0.0
        <=
        dropout_rate
        <
        1.0
    ):
        raise ValueError(
            "dropout_rate must satisfy 0 <= rate < 1."
        )

    if "regime_id" not in df.columns:
        raise ValueError(
            "regime_id is required for condition-aware imputation."
        )

    corrupted = df.copy()

    n_rows = len(
        corrupted
    )

    n_drop = int(
        round(
            dropout_rate
            *
            n_rows
        )
    )

    rng = np.random.default_rng(
        int(seed)
    )

    random_order = rng.permutation(
        n_rows
    )

    drop_positions = random_order[
        :n_drop
    ]

    drop_mask = np.zeros(
        n_rows,
        dtype=bool,
    )

    drop_mask[
        drop_positions
    ] = True

    sensor_col_pos = (
        corrupted.columns.get_loc(
            sensor
        )
    )

    corrupted.iloc[
        drop_positions,
        sensor_col_pos,
    ] = np.nan

    regime_values = (
        corrupted["regime_id"]
        .to_numpy()
    )

    imputation_values = np.empty(
        n_rows,
        dtype=np.float64,
    )

    for position_629aj in range(
        n_rows
    ):

        regime_629aj = int(
            regime_values[
                position_629aj
            ]
        )

        key_629aj = (
            regime_629aj,
            sensor,
        )

        if key_629aj not in FD003_DROPOUT_MEAN_OPERATOR_629AJ:
            raise RuntimeError(
                "Missing FD003 FIT dropout mean for "
                f"regime={regime_629aj}, sensor={sensor}."
            )

        imputation_values[
            position_629aj
        ] = (
            FD003_DROPOUT_MEAN_OPERATOR_629AJ[
                key_629aj
            ]
        )

    missing_positions = np.flatnonzero(
        drop_mask
    )

    corrupted.iloc[
        missing_positions,
        sensor_col_pos,
    ] = (
        imputation_values[
            missing_positions
        ]
    )

    return (
        corrupted,
        drop_mask,
        imputation_values,
    )


def apply_fd003_stuck_629aj(
    df,
    sensor,
    affected_fraction,
):

    validate_fd003_sensor_629aj(
        df,
        sensor,
    )

    affected_fraction = float(
        affected_fraction
    )

    if not (
        0.0
        <
        affected_fraction
        <
        1.0
    ):
        raise ValueError(
            "affected_fraction must satisfy 0 < fraction < 1."
        )

    corrupted = df.copy()

    affected_mask = np.zeros(
        len(corrupted),
        dtype=bool,
    )

    records_629aj = []

    for unit_id_629aj, group_629aj in corrupted.groupby(
        "unit_id",
        sort=False,
    ):

        group_sorted_629aj = group_629aj.sort_values(
            "cycle"
        )

        positions_629aj = group_sorted_629aj.index.to_numpy(
            dtype=np.int64
        )

        n_rows_629aj = len(
            positions_629aj
        )

        if n_rows_629aj < 2:
            raise RuntimeError(
                "Stuck corruption requires at least two cycles per engine."
            )

        n_affected_629aj = int(
            round(
                affected_fraction
                *
                n_rows_629aj
            )
        )

        n_affected_629aj = max(
            1,
            n_affected_629aj,
        )

        n_affected_629aj = min(
            n_affected_629aj,
            n_rows_629aj - 1,
        )

        onset_position_629aj = (
            n_rows_629aj
            -
            n_affected_629aj
        )

        source_position_629aj = positions_629aj[
            onset_position_629aj - 1
        ]

        affected_positions_629aj = positions_629aj[
            onset_position_629aj:
        ]

        freeze_value_629aj = float(
            corrupted.loc[
                source_position_629aj,
                sensor,
            ]
        )

        corrupted.loc[
            affected_positions_629aj,
            sensor,
        ] = freeze_value_629aj

        affected_mask[
            affected_positions_629aj
        ] = True

        records_629aj.append(
            {
                "unit_id":
                    unit_id_629aj,

                "trajectory_length":
                    n_rows_629aj,

                "requested_fraction":
                    affected_fraction,

                "affected_count":
                    n_affected_629aj,

                "actual_fraction":
                    (
                        n_affected_629aj
                        /
                        n_rows_629aj
                    ),

                "source_cycle":
                    float(
                        group_sorted_629aj.iloc[
                            onset_position_629aj - 1
                        ][
                            "cycle"
                        ]
                    ),

                "stuck_start_cycle":
                    float(
                        group_sorted_629aj.iloc[
                            onset_position_629aj
                        ][
                            "cycle"
                        ]
                    ),

                "freeze_value":
                    freeze_value_629aj,
            }
        )

    stuck_info_629aj = pd.DataFrame(
        records_629aj
    )

    return (
        corrupted,
        affected_mask,
        stuck_info_629aj,
    )


# =============================================================================
# 6) Frozen operator contract + historical-style dispatcher
# =============================================================================

FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ = {

    "bias":
        apply_fd003_bias_629aj,

    "noise":
        apply_fd003_noise_629aj,

    "drift":
        apply_fd003_drift_629aj,

    "dropout":
        apply_fd003_dropout_629aj,

    "stuck":
        apply_fd003_stuck_629aj,

    "sensor_scale":
        FD003_SENSOR_SCALE_OPERATOR_629AJ,

    "dropout_mean_reference":
        FD003_DROPOUT_MEAN_OPERATOR_629AJ,

    "base_frame_preparation":
        prepare_fd003_corruption_base_629aj,

    "sensor_order":
        FD003_OPERATOR_SENSOR_ORDER_629AJ,

    "regime_count":
        1,

    "regime_id":
        0,
}


def apply_manifest_scenario_fd003_629aj(
    base_raw,
    scenario_row,
):

    family_629aj = str(
        scenario_row[
            "family"
        ]
    )

    sensor_629aj = str(
        scenario_row[
            "sensor"
        ]
    )

    contract_629aj = (
        FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ
    )

    if family_629aj == "bias":

        corrupted_629aj, _ = (
            contract_629aj[
                "bias"
            ](
                base_raw,
                sensor_629aj,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )

    elif family_629aj == "noise":

        corrupted_629aj, _, _ = (
            contract_629aj[
                "noise"
            ](
                base_raw,
                sensor_629aj,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )

    elif family_629aj == "drift":

        corrupted_629aj, _, _ = (
            contract_629aj[
                "drift"
            ](
                base_raw,
                sensor_629aj,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )

    elif family_629aj == "dropout":

        corrupted_629aj, _, _ = (
            contract_629aj[
                "dropout"
            ](
                base_raw,
                sensor_629aj,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )

    elif family_629aj == "stuck":

        corrupted_629aj, _, _ = (
            contract_629aj[
                "stuck"
            ](
                base_raw,
                sensor_629aj,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
            )
        )

    else:

        raise ValueError(
            "Unknown corruption family: "
            +
            family_629aj
        )

    return corrupted_629aj


FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
    "apply_manifest_scenario"
] = apply_manifest_scenario_fd003_629aj


# =============================================================================
# 7) Operator implementation audit/fingerprints
# =============================================================================

operator_names_629aj = [
    "prepare_fd003_corruption_base_629aj",
    "validate_fd003_sensor_629aj",
    "apply_fd003_bias_629aj",
    "apply_fd003_noise_629aj",
    "apply_fd003_drift_629aj",
    "apply_fd003_dropout_629aj",
    "apply_fd003_stuck_629aj",
    "apply_manifest_scenario_fd003_629aj",
]

operator_rows_629aj = []

for name_629aj in operator_names_629aj:

    fn_629aj = globals()[
        name_629aj
    ]

    signature_629aj = str(
        inspect.signature(
            fn_629aj
        )
    )

    code_629aj = fn_629aj.__code__

    bytecode_sha_629aj = hashlib.sha256(
        code_629aj.co_code
    ).hexdigest()

    constants_sha_629aj = hashlib.sha256(
        repr(
            code_629aj.co_consts
        ).encode("utf-8")
    ).hexdigest()

    names_sha_629aj = hashlib.sha256(
        repr(
            code_629aj.co_names
        ).encode("utf-8")
    ).hexdigest()

    try:
        source_629aj = inspect.getsource(
            fn_629aj
        )
    except Exception:
        source_629aj = None

    source_sha_629aj = (
        hashlib.sha256(
            source_629aj.encode("utf-8")
        ).hexdigest()
        if source_629aj is not None
        else None
    )

    semantic_fingerprint_629aj = hashlib.sha256(
        json.dumps(
            {
                "signature":
                    signature_629aj,

                "bytecode_sha256":
                    bytecode_sha_629aj,

                "constants_sha256":
                    constants_sha_629aj,

                "names_sha256":
                    names_sha_629aj,
            },
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    operator_rows_629aj.append(
        {
            "name":
                name_629aj,

            "signature":
                signature_629aj,

            "source_available":
                source_629aj is not None,

            "source_sha256":
                source_sha_629aj,

            "bytecode_sha256":
                bytecode_sha_629aj,

            "semantic_sha256":
                semantic_fingerprint_629aj,
        }
    )


FD003_CORRUPTION_OPERATOR_AUDIT_629AJ = pd.DataFrame(
    operator_rows_629aj
)


# =============================================================================
# 8) Historical provenance map
# =============================================================================

historical_audit_629aj = (
    FD003_HISTORICAL_CORRUPTION_OPERATOR_SEMANTICS_AUDIT_629ZR2
    .copy()
)

historical_required_names_629aj = [
    "apply_fd002_bias_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
]

if not set(
    historical_required_names_629aj
).issubset(
    set(
        historical_audit_629aj[
            "name"
        ].astype(str)
    )
):
    raise RuntimeError(
        "Historical operator provenance audit is incomplete."
    )

historical_source_hashes_629aj = {
    str(row["name"]):
        (
            None
            if pd.isna(row["source_sha256"])
            else str(row["source_sha256"])
        )
    for _, row in historical_audit_629aj.iterrows()
    if str(row["name"]) in historical_required_names_629aj
}


# =============================================================================
# 9) Freeze operator contract metadata
# =============================================================================

operator_semantic_hash_map_629aj = {
    str(row["name"]):
        str(row["semantic_sha256"])
    for row in operator_rows_629aj
}

FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_629AJ = {

    "step":
        "6.29AJ",

    "dataset":
        "FD003",

    "historical_semantics_source":
        "exact FD002 operator implementations recovered in STEP 6.29Z-R2",

    "historical_operator_source_sha256":
        historical_source_hashes_629aj,

    "adaptation_rule":
        "preserve corruption mathematics and randomization exactly; "
        "replace only FD002 scale/imputation references with frozen "
        "FD003 FIT-only references and use a single FD003 regime_id=0",

    "base_frame_rule":
        "copy authoritative raw frame, reset index to contiguous row positions, "
        "append regime_id=0; sensor values and row order unchanged",

    "sensor_count":
        16,

    "sensor_order":
        list(
            FD003_OPERATOR_SENSOR_ORDER_629AJ
        ),

    "scale_reference_sha256":
        FD003_CORRUPTION_SCALE_REFERENCE_SHA256_629AF,

    "dropout_reference_sha256":
        FD003_DROPOUT_IMPUTATION_REFERENCE_SHA256_629AF,

    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "corruption_protocol_sha256":
        FD003_CORRUPTION_PROTOCOL_SHA256_629AF,

    "historical_manifest_semantic_contract_sha256":
        FD003_HISTORICAL_MANIFEST_SEMANTIC_CONTRACT_SHA256_629AER1,

    "operator_semantic_sha256":
        operator_semantic_hash_map_629aj,

    "bias_semantics":
        "x := x + direction * alpha * FIT_scale",

    "noise_semantics":
        "x := x + N(0,1; default_rng(seed)) * alpha * FIT_scale",

    "drift_semantics":
        "per engine, linear ramp from 0 to direction * alpha * FIT_scale",

    "dropout_semantics":
        "seeded permutation; round(rate*n_rows) rows; "
        "replace selected sensor values by FIT-only regime mean",

    "stuck_semantics":
        "per engine, round(fraction*n_rows) final rows, clipped to [1,n-1], "
        "frozen at immediately preceding sensor value",

    "manifest_dispatch_semantics":
        "family-specific dispatch using frozen manifest parameters",

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


freeze_payload_629aj = json.dumps(
    FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_629AJ,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")


FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ = hashlib.sha256(
    freeze_payload_629aj
).hexdigest()


FD003_CORRUPTION_OPERATOR_CONTRACT_LOCKED_629AJ = True
FD003_CORRUPTION_OPERATORS_CREATED_629AJ = True

FD003_CORRUPTION_EXECUTED_629AJ = False
FD003_MODEL_INFERENCE_PERFORMED_629AJ = False
FD003_MC_EXECUTION_PERFORMED_629AJ = False
FD003_DEV_SCORING_PERFORMED_629AJ = False
FD003_THRESHOLD_SELECTED_629AJ = False
FD003_POLICY_SELECTED_629AJ = False
FD003_TEST_CONTENTS_READ_629AJ = False
FD003_RUL_CONTENTS_READ_629AJ = False

all_checks_passed_629aj = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 150)

print(
    "STEP 6.29AJ — FD003 CORRUPTION-OPERATOR "
    "IMPLEMENTATION FREEZE"
)

print("=" * 150)

print("Operators created              :", True)
print("Operator contract locked       :", True)
print("Corruption executed            :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV scoring performed          :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FROZEN OPERATOR SENSOR SCOPE ---"
)

print(
    "Sensor count                   :",
    len(
        FD003_OPERATOR_SENSOR_ORDER_629AJ
    ),
)

print(
    "Sensor order                   :",
    list(
        FD003_OPERATOR_SENSOR_ORDER_629AJ
    ),
)

print(
    "\n--- OPERATOR IMPLEMENTATION AUDIT ---"
)

print(
    FD003_CORRUPTION_OPERATOR_AUDIT_629AJ.to_string(
        index=False
    )
)

print(
    "\n--- HISTORICAL SOURCE PROVENANCE ---"
)

for name_629aj in historical_required_names_629aj:
    print(
        f"{name_629aj:34s}:",
        historical_source_hashes_629aj[
            name_629aj
        ],
    )

print(
    "\nOperator-freeze contract SHA-256:",
    FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
)

print(
    "\nPASS — FD003 corruption operators are now defined and frozen "
    "with the exact recovered historical corruption semantics, using "
    "only FD003 FIT-derived scale/imputation references."
)

print(
    "No corruption has been executed. Next: run exactly one predeclared "
    "smoke scenario through corruption → normalization → windowing and "
    "deterministic inference, verify model-state safety and bridge integrity, "
    "then stop before the full 336-scenario experiment."
)

In [ ]:
# STEP 6.29AK
# FD003 ONE-SCENARIO CORRUPTION -> MODEL-BRIDGE SMOKE TEST
#
# Executes exactly one prospectively fixed smoke scenario:
#   scenario_number = 1
#   FD003_S001 = bias / sensor_2 / low / alpha=0.25 / direction=-1
#
# DEV targets/errors are NOT used.
# NO MC-dropout execution.
# NO threshold/policy selection.
# NO TEST/official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd
import torch


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aj", False)):
    raise RuntimeError("STEP 6.29AJ must pass before STEP 6.29AK.")

required_629ak = [
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_AUTHORITATIVE_DEV_RAW_629AH",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_FIT_FEATURE_MEANS_629E",
    "FD003_FIT_FEATURE_STDS_629E",
    "DEV_WINDOW_UNIT_IDS_FD003_629F",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F",
    "X_DEV_FD003_629F",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L",
    "_state_dict_sha256_629k",
]

missing_629ak = [name for name in required_629ak if name not in globals()]
if missing_629ak:
    raise RuntimeError(
        "Missing required STEP 6.29AK object(s): " + ", ".join(missing_629ak)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AJ", True)):
    raise RuntimeError("Unexpected TEST-access state before STEP 6.29AK.")

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AJ", True)):
    raise RuntimeError("Unexpected official-RUL-access state before STEP 6.29AK.")

if bool(globals().get("FD003_SMOKE_SCENARIO_COMPLETE_629AK", False)):
    raise RuntimeError("STEP 6.29AK has already been completed.")


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629ak(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.29AK hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _manifest_sha256_629ak(df):
    columns = [
        "scenario_id",
        "scenario_number",
        "family",
        "sensor",
        "severity",
        "alpha",
        "direction",
        "requested_fraction",
        "seed",
    ]

    records = []
    for row in df[columns].itertuples(index=False, name=None):
        out = []
        for value in row:
            if isinstance(value, np.generic):
                value = value.item()
            if value is None or (
                isinstance(value, float) and np.isnan(value)
            ):
                out.append(None)
            elif isinstance(value, (str, bool, int, float)):
                out.append(value)
            else:
                raise TypeError(
                    "Unsupported manifest scalar type during STEP 6.29AK."
                )
        records.append(out)

    payload = json.dumps(
        records,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


# =============================================================================
# 3) Verify manifest and freeze the predeclared smoke scenario
# =============================================================================

manifest_629ak = FD003_CORRUPTION_MANIFEST_629AF

if _manifest_sha256_629ak(manifest_629ak) != FD003_CORRUPTION_MANIFEST_SHA256_629AF:
    raise RuntimeError("Frozen FD003 corruption manifest changed before smoke execution.")

smoke_rows_629ak = manifest_629ak[
    manifest_629ak["scenario_number"] == 1
]

if len(smoke_rows_629ak) != 1:
    raise RuntimeError("scenario_number=1 is not unique.")

smoke_row_629ak = smoke_rows_629ak.iloc[0]

if str(smoke_row_629ak["scenario_id"]) != "FD003_S001":
    raise RuntimeError("Unexpected smoke scenario ID.")
if str(smoke_row_629ak["family"]) != "bias":
    raise RuntimeError("Unexpected smoke family.")
if str(smoke_row_629ak["sensor"]) != "sensor_2":
    raise RuntimeError("Unexpected smoke sensor.")
if str(smoke_row_629ak["severity"]) != "low":
    raise RuntimeError("Unexpected smoke severity.")
if float(smoke_row_629ak["alpha"]) != 0.25:
    raise RuntimeError("Unexpected smoke alpha.")
if int(smoke_row_629ak["direction"]) != -1:
    raise RuntimeError("Unexpected smoke direction.")

FD003_SMOKE_SCENARIO_CONTRACT_629AK = {
    "step": "6.29AK",
    "selection_rule": "first row of the already-frozen manifest; no outcome-based selection",
    "scenario_number": 1,
    "scenario_id": "FD003_S001",
    "family": "bias",
    "sensor": "sensor_2",
    "severity": "low",
    "alpha": 0.25,
    "direction": -1,
    "dev_targets_used": False,
    "dev_errors_used": False,
    "mc_execution": False,
    "threshold_selection": False,
    "policy_selection": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_SMOKE_SCENARIO_CONTRACT_SHA256_629AK = hashlib.sha256(
    json.dumps(
        FD003_SMOKE_SCENARIO_CONTRACT_629AK,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


# =============================================================================
# 4) Execute exactly one corruption
# =============================================================================

base_raw_629ak = FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
    "base_frame_preparation"
](FD003_AUTHORITATIVE_DEV_RAW_629AH)

if base_raw_629ak.shape != (5286, 27):
    raise RuntimeError("Prepared DEV corruption base shape is not (5286, 27).")

corrupted_raw_629ak = FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
    "apply_manifest_scenario"
](
    base_raw_629ak,
    smoke_row_629ak,
)

if corrupted_raw_629ak.shape != base_raw_629ak.shape:
    raise RuntimeError("Smoke corruption changed frame shape.")
if list(corrupted_raw_629ak.columns) != list(base_raw_629ak.columns):
    raise RuntimeError("Smoke corruption changed frame columns.")
if not corrupted_raw_629ak.index.equals(base_raw_629ak.index):
    raise RuntimeError("Smoke corruption changed frame index.")


# =============================================================================
# 5) Raw corruption semantics audit
# =============================================================================

target_sensor_629ak = "sensor_2"

expected_bias_629ak = float(
    -0.25
    * FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
        "sensor_scale"
    ][target_sensor_629ak]
)

actual_delta_629ak = (
    corrupted_raw_629ak[target_sensor_629ak].to_numpy(dtype=np.float64)
    -
    base_raw_629ak[target_sensor_629ak].to_numpy(dtype=np.float64)
)

raw_bias_exact_629ak = bool(
    np.allclose(
        actual_delta_629ak,
        expected_bias_629ak,
        rtol=1e-12,
        atol=1e-12,
    )
)

if not raw_bias_exact_629ak:
    raise RuntimeError("Smoke bias does not match the frozen operator semantics.")

non_target_columns_629ak = [
    c for c in base_raw_629ak.columns
    if c != target_sensor_629ak
]

non_target_raw_exact_629ak = bool(
    base_raw_629ak[non_target_columns_629ak].equals(
        corrupted_raw_629ak[non_target_columns_629ak]
    )
)

if not non_target_raw_exact_629ak:
    raise RuntimeError("Smoke corruption altered a non-target raw column.")


# =============================================================================
# 6) Frozen normalization
# =============================================================================

raw_features_629ak = [str(x) for x in FD003_SELECTED_RAW_FEATURES_629E]
norm_features_629ak = [str(x) for x in FD003_NORMALIZED_FEATURE_NAMES_629E]

means_629ak = np.asarray(
    FD003_FIT_FEATURE_MEANS_629E,
    dtype=np.float64,
).reshape(1, -1)

stds_629ak = np.asarray(
    FD003_FIT_FEATURE_STDS_629E,
    dtype=np.float64,
).reshape(1, -1)

raw_matrix_629ak = corrupted_raw_629ak[
    raw_features_629ak
].to_numpy(dtype=np.float64)

FD003_SMOKE_DEV_Z_629AK = (
    (raw_matrix_629ak - means_629ak) / stds_629ak
).astype(np.float64, copy=False)

if FD003_SMOKE_DEV_Z_629AK.shape != (5286, 18):
    raise RuntimeError("Smoke normalized matrix shape is not (5286, 18).")
if not np.isfinite(FD003_SMOKE_DEV_Z_629AK).all():
    raise RuntimeError("Smoke normalized matrix contains non-finite values.")

FD003_SMOKE_DEV_Z_SHA256_629AK = _raw_sha256_629ak(
    FD003_SMOKE_DEV_Z_629AK
)


# =============================================================================
# 7) Rebuild smoke windows
# =============================================================================

smoke_frame_629ak = corrupted_raw_629ak[
    ["unit_id", "cycle"]
].copy()

for j, feature in enumerate(norm_features_629ak):
    smoke_frame_629ak[feature] = FD003_SMOKE_DEV_Z_629AK[:, j]

window_blocks_629ak = []
window_units_629ak = []
window_cycles_629ak = []

for unit_id, group in smoke_frame_629ak.groupby("unit_id", sort=False):
    group = group.sort_values("cycle", kind="mergesort")
    values = group[norm_features_629ak].to_numpy(dtype=np.float64)
    cycles = group["cycle"].to_numpy(dtype=np.int64)

    for start in range(0, len(group) - 30 + 1):
        stop = start + 30

        window_blocks_629ak.append(
            values[start:stop, :]
        )
        window_units_629ak.append(
            int(unit_id)
        )
        window_cycles_629ak.append(
            int(cycles[stop - 1])
        )

FD003_SMOKE_DEV_WINDOWS_629AK = np.asarray(
    window_blocks_629ak,
    dtype=np.float32,
)

FD003_SMOKE_DEV_WINDOW_UNIT_IDS_629AK = np.asarray(
    window_units_629ak,
    dtype=np.int64,
)

FD003_SMOKE_DEV_ENDPOINT_CYCLES_629AK = np.asarray(
    window_cycles_629ak,
    dtype=np.int64,
)

if FD003_SMOKE_DEV_WINDOWS_629AK.shape != (4706, 30, 18):
    raise RuntimeError("Smoke window shape is not (4706, 30, 18).")

membership_exact_629ak = bool(
    np.array_equal(
        FD003_SMOKE_DEV_WINDOW_UNIT_IDS_629AK,
        np.asarray(
            DEV_WINDOW_UNIT_IDS_FD003_629F,
            dtype=np.int64,
        ),
    )
)

endpoint_exact_629ak = bool(
    np.array_equal(
        FD003_SMOKE_DEV_ENDPOINT_CYCLES_629AK,
        np.asarray(
            DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
            dtype=np.int64,
        ),
    )
)

if not membership_exact_629ak:
    raise RuntimeError("Smoke window engine membership changed.")
if not endpoint_exact_629ak:
    raise RuntimeError("Smoke window endpoint cycles changed.")


# =============================================================================
# 8) Verify only the intended model feature changed
# =============================================================================

clean_windows_629ak = np.asarray(
    X_DEV_FD003_629F,
    dtype=np.float32,
)

target_feature_index_629ak = raw_features_629ak.index(
    target_sensor_629ak
)

non_target_indices_629ak = [
    i
    for i in range(len(raw_features_629ak))
    if i != target_feature_index_629ak
]

non_target_window_exact_629ak = bool(
    np.array_equal(
        FD003_SMOKE_DEV_WINDOWS_629AK[:, :, non_target_indices_629ak],
        clean_windows_629ak[:, :, non_target_indices_629ak],
    )
)

if not non_target_window_exact_629ak:
    raise RuntimeError("Smoke corruption changed a non-target model feature.")

target_delta_629ak = (
    FD003_SMOKE_DEV_WINDOWS_629AK[
        :, :, target_feature_index_629ak
    ].astype(np.float64)
    -
    clean_windows_629ak[
        :, :, target_feature_index_629ak
    ].astype(np.float64)
)

if not np.any(target_delta_629ak != 0.0):
    raise RuntimeError("Smoke corruption produced no target-feature change.")

expected_normalized_shift_629ak = float(
    expected_bias_629ak
    /
    float(
        FD003_FIT_FEATURE_STDS_629E[
            target_feature_index_629ak
        ]
    )
)

target_shift_min_629ak = float(target_delta_629ak.min())
target_shift_mean_629ak = float(target_delta_629ak.mean())
target_shift_max_629ak = float(target_delta_629ak.max())

FD003_SMOKE_DEV_WINDOW_SHA256_629AK = _raw_sha256_629ak(
    FD003_SMOKE_DEV_WINDOWS_629AK
)


# =============================================================================
# 9) Verify frozen clean prediction identity
# =============================================================================

clean_prediction_629ak = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)

expected_clean_prediction_sha_629ak = str(
    FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L[
        "prediction_sha256"
    ]
)

if (
    _raw_sha256_629ak(clean_prediction_629ak)
    !=
    expected_clean_prediction_sha_629ak
):
    raise RuntimeError("Frozen clean DEV prediction fingerprint changed.")


# =============================================================================
# 10) Deterministic inference on the smoke scenario only
# =============================================================================

model_629ak = FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH

model_sha_before_629ak = _state_dict_sha256_629k(
    model_629ak
)

if (
    model_sha_before_629ak
    !=
    FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
):
    raise RuntimeError("FD003 model state changed before smoke inference.")

model_mode_before_629ak = bool(
    model_629ak.training
)

model_device_629ak = next(
    model_629ak.parameters()
).device

pred_parts_629ak = []

try:
    model_629ak.eval()

    with torch.no_grad():
        for start in range(
            0,
            len(FD003_SMOKE_DEV_WINDOWS_629AK),
            512,
        ):
            stop = min(
                start + 512,
                len(FD003_SMOKE_DEV_WINDOWS_629AK),
            )

            xb = torch.from_numpy(
                FD003_SMOKE_DEV_WINDOWS_629AK[
                    start:stop
                ]
            ).to(
                device=model_device_629ak,
                dtype=torch.float32,
            )

            pred = model_629ak(
                xb
            )

            pred_parts_629ak.append(
                pred.detach()
                .cpu()
                .numpy()
                .astype(np.float64, copy=False)
                .reshape(-1)
            )

finally:
    model_629ak.train(
        model_mode_before_629ak
    )

FD003_SMOKE_DEV_PREDICTION_629AK = np.concatenate(
    pred_parts_629ak
)

if FD003_SMOKE_DEV_PREDICTION_629AK.shape != (4706,):
    raise RuntimeError("Smoke prediction shape is not (4706,).")
if not np.isfinite(FD003_SMOKE_DEV_PREDICTION_629AK).all():
    raise RuntimeError("Smoke prediction contains non-finite values.")

model_sha_after_629ak = _state_dict_sha256_629k(
    model_629ak
)

model_unchanged_629ak = bool(
    model_sha_after_629ak
    ==
    model_sha_before_629ak
)

model_mode_restored_629ak = bool(
    bool(model_629ak.training)
    ==
    model_mode_before_629ak
)

if not model_unchanged_629ak:
    raise RuntimeError("FD003 model state changed during smoke inference.")
if not model_mode_restored_629ak:
    raise RuntimeError("FD003 model mode was not restored after smoke inference.")


# =============================================================================
# 11) Label-free prediction-response diagnostics
# =============================================================================

prediction_delta_629ak = (
    FD003_SMOKE_DEV_PREDICTION_629AK
    -
    clean_prediction_629ak
)

prediction_changed_count_629ak = int(
    np.count_nonzero(
        prediction_delta_629ak != 0.0
    )
)

mean_abs_prediction_delta_629ak = float(
    np.mean(
        np.abs(
            prediction_delta_629ak
        )
    )
)

max_abs_prediction_delta_629ak = float(
    np.max(
        np.abs(
            prediction_delta_629ak
        )
    )
)

if prediction_changed_count_629ak <= 0:
    raise RuntimeError(
        "Smoke corruption produced no deterministic prediction response."
    )

FD003_SMOKE_DEV_PREDICTION_SHA256_629AK = _raw_sha256_629ak(
    FD003_SMOKE_DEV_PREDICTION_629AK
)


# =============================================================================
# 12) Read-only evidence + contract
# =============================================================================

for arr in (
    FD003_SMOKE_DEV_Z_629AK,
    FD003_SMOKE_DEV_WINDOWS_629AK,
    FD003_SMOKE_DEV_WINDOW_UNIT_IDS_629AK,
    FD003_SMOKE_DEV_ENDPOINT_CYCLES_629AK,
    FD003_SMOKE_DEV_PREDICTION_629AK,
):
    arr.setflags(write=False)

FD003_SMOKE_BRIDGE_EVIDENCE_629AK = {
    "step": "6.29AK",
    "smoke_scenario_contract_sha256":
        FD003_SMOKE_SCENARIO_CONTRACT_SHA256_629AK,
    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
    "scenario_number": 1,
    "scenario_id": "FD003_S001",
    "expected_raw_bias_amount": expected_bias_629ak,
    "raw_bias_exact": raw_bias_exact_629ak,
    "non_target_raw_columns_exact": non_target_raw_exact_629ak,
    "normalized_sha256": FD003_SMOKE_DEV_Z_SHA256_629AK,
    "window_sha256": FD003_SMOKE_DEV_WINDOW_SHA256_629AK,
    "window_membership_exact": membership_exact_629ak,
    "window_endpoint_cycles_exact": endpoint_exact_629ak,
    "non_target_window_features_exact": non_target_window_exact_629ak,
    "target_feature_index": int(target_feature_index_629ak),
    "expected_normalized_shift": expected_normalized_shift_629ak,
    "target_shift_min": target_shift_min_629ak,
    "target_shift_mean": target_shift_mean_629ak,
    "target_shift_max": target_shift_max_629ak,
    "prediction_sha256": FD003_SMOKE_DEV_PREDICTION_SHA256_629AK,
    "prediction_changed_count": prediction_changed_count_629ak,
    "mean_abs_prediction_delta": mean_abs_prediction_delta_629ak,
    "max_abs_prediction_delta": max_abs_prediction_delta_629ak,
    "dev_targets_used": False,
    "dev_errors_used": False,
    "mc_execution_performed": False,
    "model_state_before": model_sha_before_629ak,
    "model_state_after": model_sha_after_629ak,
    "model_state_unchanged": model_unchanged_629ak,
    "model_mode_restored": model_mode_restored_629ak,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_SMOKE_BRIDGE_EVIDENCE_SHA256_629AK = hashlib.sha256(
    json.dumps(
        FD003_SMOKE_BRIDGE_EVIDENCE_629AK,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

FD003_SMOKE_SCENARIO_COMPLETE_629AK = True
FD003_CORRUPTION_EXECUTED_629AK = True
FD003_MODEL_INFERENCE_PERFORMED_629AK = True
FD003_MC_EXECUTION_PERFORMED_629AK = False
FD003_DEV_SCORING_PERFORMED_629AK = False
FD003_THRESHOLD_SELECTED_629AK = False
FD003_POLICY_SELECTED_629AK = False
FD003_TEST_CONTENTS_READ_629AK = False
FD003_RUL_CONTENTS_READ_629AK = False

all_checks_passed_629ak = True


# =============================================================================
# 13) Output
# =============================================================================

print("=" * 152)
print(
    "STEP 6.29AK — FD003 ONE-SCENARIO "
    "CORRUPTION → MODEL-BRIDGE SMOKE TEST"
)
print("=" * 152)

print("Smoke scenario executed        :", True)
print("DEV targets used               :", False)
print("DEV error scoring performed    :", False)
print("Deterministic inference run    :", True)
print("MC-dropout execution performed :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- PREDECLARED SMOKE SCENARIO ---")
print("Scenario number                :", 1)
print("Scenario ID                    :", "FD003_S001")
print(
    "Family / sensor / severity     :",
    ("bias", "sensor_2", "low"),
)
print(
    "Alpha / direction              :",
    (0.25, -1),
)
print(
    "Smoke-scenario contract SHA-256:",
    FD003_SMOKE_SCENARIO_CONTRACT_SHA256_629AK,
)

print("\n--- RAW CORRUPTION AUDIT ---")
print(
    "Expected constant bias         :",
    expected_bias_629ak,
)
print(
    "Raw bias exact                 :",
    raw_bias_exact_629ak,
)
print(
    "Non-target raw columns exact   :",
    non_target_raw_exact_629ak,
)

print("\n--- WINDOW BRIDGE AUDIT ---")
print(
    "Smoke window shape             :",
    FD003_SMOKE_DEV_WINDOWS_629AK.shape,
)
print(
    "Engine membership exact        :",
    membership_exact_629ak,
)
print(
    "Endpoint cycles exact          :",
    endpoint_exact_629ak,
)
print(
    "Non-target features exact      :",
    non_target_window_exact_629ak,
)
print(
    "Target model feature index     :",
    target_feature_index_629ak,
)
print(
    "Expected normalized shift      :",
    expected_normalized_shift_629ak,
)
print(
    "Observed target shift min/mean/max:",
    (
        target_shift_min_629ak,
        target_shift_mean_629ak,
        target_shift_max_629ak,
    ),
)
print(
    "Smoke window SHA-256           :",
    FD003_SMOKE_DEV_WINDOW_SHA256_629AK,
)

print("\n--- DETERMINISTIC MODEL RESPONSE ---")
print(
    "Prediction shape               :",
    FD003_SMOKE_DEV_PREDICTION_629AK.shape,
)
print(
    "Prediction changed count       :",
    prediction_changed_count_629ak,
)
print(
    "Mean |prediction delta|        :",
    mean_abs_prediction_delta_629ak,
)
print(
    "Max |prediction delta|         :",
    max_abs_prediction_delta_629ak,
)
print(
    "Smoke prediction SHA-256       :",
    FD003_SMOKE_DEV_PREDICTION_SHA256_629AK,
)

print("\n--- MODEL SAFETY ---")
print("Model SHA before               :", model_sha_before_629ak)
print("Model SHA after                :", model_sha_after_629ak)
print("Model state unchanged          :", model_unchanged_629ak)
print("Model mode restored            :", model_mode_restored_629ak)

print(
    "\nSmoke-evidence SHA-256         :",
    FD003_SMOKE_BRIDGE_EVIDENCE_SHA256_629AK,
)

print(
    "\nPASS — one predeclared FD003 corruption scenario completed the "
    "raw→normalization→window→deterministic-inference bridge safely."
)

print(
    "No DEV target/error, MC-dropout, threshold/policy selection, TEST, "
    "or official RUL was used. Next: inspect this evidence before freezing "
    "the full 336-scenario corruption-execution protocol."
)

In [ ]:
# STEP 6.29AL
# FD003 LABEL-FREE CORRUPTION-RISK BRIDGE SOURCE LOCATOR
#
# Metadata-only locator for the already-frozen reliability pipeline required
# by the full corruption experiment.
#
# NO corruption execution.
# NO model inference / MC execution.
# NO DEV labels/errors.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import inspect
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ak", False)):
    raise RuntimeError(
        "STEP 6.29AK must pass before STEP 6.29AL."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AK", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AL."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AK", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AL."
    )


required_known_629al = [
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_DEV_G1_RISK_629S",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
]

missing_known_629al = [
    name
    for name in required_known_629al
    if name not in globals()
]

if missing_known_629al:
    raise RuntimeError(
        "Missing required frozen reliability/corruption object(s): "
        + ", ".join(missing_known_629al)
    )


# =============================================================================
# 2) Strict metadata-only search scope
# =============================================================================

step_tokens_629al = (
    "629m",
    "629o",
    "629q",
    "629r",
    "629s",
    "629t",
    "629u",
    "629v",
)

reliability_tokens_629al = (
    "mc",
    "phase",
    "uncert",
    "g1",
    "g2",
    "risk",
    "midrank",
    "reference",
)

forbidden_tokens_629al = (
    "test",
    "official",
    "rul_fd",
    "target",
    "y_dev",
    "abs_error",
    "error",
    "score",
    "threshold",
    "policy",
    "accept",
    "abstain",
)


candidate_names_629al = []

for name_629al in sorted(globals().keys()):

    lower_629al = name_629al.lower()

    if (
        "fd003" not in lower_629al
        and
        not lower_629al.startswith("apply_fd003")
    ):
        continue

    if any(
        token_629al in lower_629al
        for token_629al in forbidden_tokens_629al
    ):
        continue

    if not any(
        token_629al in lower_629al
        for token_629al in reliability_tokens_629al
    ):
        continue

    if not any(
        token_629al in lower_629al
        for token_629al in step_tokens_629al
    ):
        continue

    candidate_names_629al.append(
        name_629al
    )


# =============================================================================
# 3) Metadata inventory only
# =============================================================================

rows_629al = []

for name_629al in candidate_names_629al:

    obj_629al = globals()[
        name_629al
    ]

    row_629al = {
        "name":
            name_629al,

        "type":
            type(
                obj_629al
            ).__name__,

        "shape_or_len":
            None,

        "dtype":
            None,

        "keys_or_columns":
            None,

        "signature":
            None,
    }

    if isinstance(
        obj_629al,
        np.ndarray,
    ):

        row_629al[
            "shape_or_len"
        ] = tuple(
            int(x)
            for x in obj_629al.shape
        )

        row_629al[
            "dtype"
        ] = str(
            obj_629al.dtype
        )

    elif isinstance(
        obj_629al,
        pd.DataFrame,
    ):

        row_629al[
            "shape_or_len"
        ] = tuple(
            int(x)
            for x in obj_629al.shape
        )

        row_629al[
            "keys_or_columns"
        ] = [
            str(c)
            for c in obj_629al.columns
        ]

    elif isinstance(
        obj_629al,
        dict,
    ):

        row_629al[
            "shape_or_len"
        ] = int(
            len(
                obj_629al
            )
        )

        row_629al[
            "keys_or_columns"
        ] = sorted(
            str(k)
            for k in obj_629al.keys()
        )

    elif isinstance(
        obj_629al,
        (list, tuple),
    ):

        row_629al[
            "shape_or_len"
        ] = int(
            len(
                obj_629al
            )
        )

    elif callable(
        obj_629al
    ):

        try:
            row_629al[
                "signature"
            ] = str(
                inspect.signature(
                    obj_629al
                )
            )

        except Exception:
            row_629al[
                "signature"
            ] = "<unavailable>"

    else:

        try:
            row_629al[
                "shape_or_len"
            ] = int(
                len(
                    obj_629al
                )
            )

        except Exception:
            row_629al[
                "shape_or_len"
            ] = "<scalar_or_uninspected>"

    rows_629al.append(
        row_629al
    )


FD003_CORRUPTION_RISK_BRIDGE_SOURCE_LOCATOR_629AL = pd.DataFrame(
    rows_629al
)


# =============================================================================
# 4) Safety state
# =============================================================================

FD003_CORRUPTION_EXECUTED_629AL = False
FD003_MODEL_INFERENCE_PERFORMED_629AL = False
FD003_MC_EXECUTION_PERFORMED_629AL = False
FD003_DEV_SCORING_PERFORMED_629AL = False
FD003_THRESHOLD_SELECTED_629AL = False
FD003_POLICY_SELECTED_629AL = False
FD003_TEST_CONTENTS_READ_629AL = False
FD003_RUL_CONTENTS_READ_629AL = False

all_checks_passed_629al = True


# =============================================================================
# 5) Output
# =============================================================================

print("=" * 152)

print(
    "STEP 6.29AL — FD003 LABEL-FREE CORRUPTION-RISK "
    "BRIDGE SOURCE LOCATOR"
)

print("=" * 152)

print("Additional corruption executed :", False)
print("Model inference performed      :", False)
print("MC-dropout execution performed :", False)
print("DEV labels/errors used         :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- TARGETED FROZEN RELIABILITY-BRIDGE INVENTORY ---"
)

if len(
    FD003_CORRUPTION_RISK_BRIDGE_SOURCE_LOCATOR_629AL
) == 0:

    print(
        "<no targeted reliability-bridge objects found>"
    )

else:

    print(
        FD003_CORRUPTION_RISK_BRIDGE_SOURCE_LOCATOR_629AL.to_string(
            index=False
        )
    )

print(
    "\nPASS — metadata-only locator complete. "
    "No new corruption, inference, MC execution, DEV outcome/error use, "
    "threshold/policy selection, TEST, or official RUL access occurred."
)

print(
    "Next: lock the exact MC-dropout → phase adjustment → G1 → G2 "
    "label-free risk bridge, then freeze the full 336-scenario execution protocol."
)

In [ ]:
# STEP 6.29AM
# FD003 LABEL-FREE RISK-BRIDGE CERTIFICATION
# + FULL CORRUPTION EXECUTION-PROTOCOL FREEZE
#
# Uses only already-existing clean DEV prediction/MC evidence to certify:
#
# deterministic prediction
#       -> MC std
#       -> phase adjustment
#       -> G1
#       -> G2 final selective risk
#
# Then freezes the prospective execution protocol for all 336 corruptions.
#
# NO new corruption.
# NO new model inference.
# NO new MC execution.
# NO DEV labels/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.

import hashlib
import inspect
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(
    globals().get(
        "all_checks_passed_629al",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AL must pass before STEP 6.29AM."
    )


required_629am = [
    "FD003_CLEAN_DEV_PREDICTION_629L",

    "FD003_DEV_MC_STD_629M",
    "FD003_DEV_MC_STD_SHA256_629M",

    "FD003_MC_PROTOCOL_629M",
    "FD003_MC_PROTOCOL_SHA256_629M",

    "FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_629Q",
    "FD003_FIT_UNCERTAINTY_CALIBRATION_PROTOCOL_SHA256_629Q",

    "FD003_PHASE_EDGES_629R",
    "FD003_PHASE_EDGES_SHA256_629R",
    "FD003_PHASE_REFERENCES_629R",
    "FD003_PHASE_REFERENCE_SHA256_629R",

    "FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S",
    "FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S",

    "FD003_DEV_PHASE_ID_629S",
    "FD003_DEV_PHASE_ID_SHA256_629S",

    "FD003_G1_REFERENCE_629S",
    "FD003_G1_REFERENCE_SHA256_629S",

    "FD003_CLEAN_DEV_G1_RISK_629S",
    "FD003_CLEAN_DEV_G1_RISK_SHA256_629S",

    "FD003_G2_UNIQUE_VALUES_629V",
    "FD003_G2_UNIQUE_VALUES_SHA256_629V",

    "FD003_G2_MID_MASS_629V",
    "FD003_G2_MID_MASS_SHA256_629V",

    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V",

    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",

    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_SHA256_629AI",

    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",

    "mc_dropout_predict_fd003_629m",
    "apply_fd003_phase_adjustment_629s",
    "apply_fd003_g1_right_ecdf_629s",
    "apply_fd003_g2_weighted_midrank_629v",
]


missing_629am = [
    name
    for name in required_629am
    if name not in globals()
]

if missing_629am:
    raise RuntimeError(
        "Missing required STEP 6.29AM object(s): "
        + ", ".join(missing_629am)
    )


if bool(
    globals().get(
        "FD003_TEST_CONTENTS_READ_629AL",
        True,
    )
):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AM."
    )


if bool(
    globals().get(
        "FD003_RUL_CONTENTS_READ_629AL",
        True,
    )
):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AM."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _raw_sha256_629am(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AM hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _function_fingerprint_629am(fn):

    signature = str(
        inspect.signature(fn)
    )

    code = fn.__code__

    bytecode_sha = hashlib.sha256(
        code.co_code
    ).hexdigest()

    constants_sha = hashlib.sha256(
        repr(
            code.co_consts
        ).encode("utf-8")
    ).hexdigest()

    names_sha = hashlib.sha256(
        repr(
            code.co_names
        ).encode("utf-8")
    ).hexdigest()

    try:
        source = inspect.getsource(
            fn
        )
    except Exception:
        source = None

    source_sha = (
        hashlib.sha256(
            source.encode("utf-8")
        ).hexdigest()
        if source is not None
        else None
    )

    semantic_sha = hashlib.sha256(
        json.dumps(
            {
                "signature":
                    signature,

                "bytecode_sha256":
                    bytecode_sha,

                "constants_sha256":
                    constants_sha,

                "names_sha256":
                    names_sha,
            },
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    return {
        "signature":
            signature,

        "source_sha256":
            source_sha,

        "bytecode_sha256":
            bytecode_sha,

        "semantic_sha256":
            semantic_sha,
    }


def _candidate_arrays_629am(result):

    if isinstance(
        result,
        tuple,
    ):
        return list(result)

    if isinstance(
        result,
        list,
    ):
        return list(result)

    return [result]


def _extract_by_hash_629am(
    result,
    expected_sha,
    expected_shape,
    label,
):

    matches = []

    for item in _candidate_arrays_629am(
        result
    ):

        try:
            arr = np.asarray(
                item
            )
        except Exception:
            continue

        if (
            arr.shape
            !=
            tuple(expected_shape)
        ):
            continue

        if arr.dtype.hasobject:
            continue

        if (
            _raw_sha256_629am(
                arr
            )
            ==
            expected_sha
        ):
            matches.append(
                arr
            )

    if len(matches) != 1:
        raise RuntimeError(
            f"{label}: expected exactly one helper output matching "
            f"frozen evidence; found {len(matches)}."
        )

    return matches[0]


# =============================================================================
# 3) Verify canonical MC convention
# =============================================================================

if int(
    FD003_MC_PROTOCOL_629M[
        "passes"
    ]
) != 50:
    raise RuntimeError(
        "Canonical MC pass count changed."
    )


if int(
    FD003_MC_PROTOCOL_629M[
        "seed"
    ]
) != 42:
    raise RuntimeError(
        "Canonical MC seed changed."
    )


if int(
    FD003_MC_PROTOCOL_629M[
        "batch_size"
    ]
) != 512:
    raise RuntimeError(
        "Canonical MC batch size changed."
    )


if int(
    FD003_MC_PROTOCOL_629M[
        "std_ddof"
    ]
) != 0:
    raise RuntimeError(
        "Canonical MC std_ddof changed."
    )


# =============================================================================
# 4) Frozen clean inputs
# =============================================================================

clean_pred_629am = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)


mc_std_629am = np.asarray(
    FD003_DEV_MC_STD_629M,
    dtype=np.float64,
).reshape(-1)


if clean_pred_629am.shape != (4706,):
    raise RuntimeError(
        "Frozen clean prediction shape changed."
    )


if mc_std_629am.shape != (4706,):
    raise RuntimeError(
        "Frozen clean MC-STD shape changed."
    )


if (
    _raw_sha256_629am(
        mc_std_629am
    )
    !=
    FD003_DEV_MC_STD_SHA256_629M
):
    raise RuntimeError(
        "Frozen clean DEV MC-STD fingerprint changed."
    )


# =============================================================================
# 5) Certify phase-adjustment helper
# =============================================================================

phase_result_629am = (
    apply_fd003_phase_adjustment_629s(
        clean_pred_629am,
        mc_std_629am,
    )
)


phase_adjusted_rebuilt_629am = _extract_by_hash_629am(
    phase_result_629am,
    FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S,
    (4706,),
    "phase adjustment",
)


phase_id_helper_match_629am = False

for item in _candidate_arrays_629am(
    phase_result_629am
):

    try:
        arr = np.asarray(
            item
        )
    except Exception:
        continue

    if (
        arr.shape == (4706,)
        and
        not arr.dtype.hasobject
    ):

        if (
            _raw_sha256_629am(
                arr
            )
            ==
            FD003_DEV_PHASE_ID_SHA256_629S
        ):
            phase_id_helper_match_629am = True


if not np.array_equal(
    np.asarray(
        phase_adjusted_rebuilt_629am,
        dtype=np.float64,
    ),
    np.asarray(
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_629S,
        dtype=np.float64,
    ),
):
    raise RuntimeError(
        "Phase-adjusted uncertainty is not elementwise exact."
    )


# =============================================================================
# 6) Certify G1 helper
# =============================================================================

g1_result_629am = (
    apply_fd003_g1_right_ecdf_629s(
        phase_adjusted_rebuilt_629am
    )
)


g1_rebuilt_629am = _extract_by_hash_629am(
    g1_result_629am,
    FD003_CLEAN_DEV_G1_RISK_SHA256_629S,
    (4706,),
    "G1 transform",
)


if not np.array_equal(
    np.asarray(
        g1_rebuilt_629am,
        dtype=np.float64,
    ),
    np.asarray(
        FD003_CLEAN_DEV_G1_RISK_629S,
        dtype=np.float64,
    ),
):
    raise RuntimeError(
        "G1 risk is not elementwise exact."
    )


# =============================================================================
# 7) Certify G2 helper / final selective risk
# =============================================================================

g2_result_629am = (
    apply_fd003_g2_weighted_midrank_629v(
        g1_rebuilt_629am
    )
)


final_risk_rebuilt_629am = _extract_by_hash_629am(
    g2_result_629am,
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,
    (4706,),
    "G2 transform",
)


if not np.array_equal(
    np.asarray(
        final_risk_rebuilt_629am,
        dtype=np.float64,
    ),
    np.asarray(
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
        dtype=np.float64,
    ),
):
    raise RuntimeError(
        "Final selective risk is not elementwise exact."
    )


# =============================================================================
# 8) Verify frozen calibration references unchanged
# =============================================================================

if (
    _raw_sha256_629am(
        FD003_PHASE_EDGES_629R
    )
    !=
    FD003_PHASE_EDGES_SHA256_629R
):
    raise RuntimeError(
        "Phase-edge fingerprint changed."
    )


for phase_id_629am, ref_629am in (
    FD003_PHASE_REFERENCES_629R.items()
):

    expected_sha_629am = (
        FD003_PHASE_REFERENCE_SHA256_629R[
            phase_id_629am
        ]
    )

    if (
        _raw_sha256_629am(
            ref_629am
        )
        !=
        expected_sha_629am
    ):
        raise RuntimeError(
            "Phase-reference fingerprint changed for phase "
            + str(phase_id_629am)
        )


if (
    _raw_sha256_629am(
        FD003_G1_REFERENCE_629S
    )
    !=
    FD003_G1_REFERENCE_SHA256_629S
):
    raise RuntimeError(
        "G1 reference fingerprint changed."
    )


if (
    _raw_sha256_629am(
        FD003_G2_UNIQUE_VALUES_629V
    )
    !=
    FD003_G2_UNIQUE_VALUES_SHA256_629V
):
    raise RuntimeError(
        "G2 unique-value fingerprint changed."
    )


if (
    _raw_sha256_629am(
        FD003_G2_MID_MASS_629V
    )
    !=
    FD003_G2_MID_MASS_SHA256_629V
):
    raise RuntimeError(
        "G2 mid-mass fingerprint changed."
    )


# =============================================================================
# 9) Freeze risk-bridge function fingerprints
# =============================================================================

function_map_629am = {

    "mc_dropout_predict_fd003_629m":
        mc_dropout_predict_fd003_629m,

    "apply_fd003_phase_adjustment_629s":
        apply_fd003_phase_adjustment_629s,

    "apply_fd003_g1_right_ecdf_629s":
        apply_fd003_g1_right_ecdf_629s,

    "apply_fd003_g2_weighted_midrank_629v":
        apply_fd003_g2_weighted_midrank_629v,
}


FD003_CORRUPTION_RISK_BRIDGE_FUNCTION_AUDIT_629AM = {

    name:
        _function_fingerprint_629am(
            fn
        )

    for name, fn in function_map_629am.items()
}


# =============================================================================
# 10) Freeze prospective full 336-scenario execution protocol
# =============================================================================

FD003_CORRUPTION_EXECUTION_PROTOCOL_629AM = {

    "step":
        "6.29AM",

    "dataset":
        "FD003",

    "scenario_manifest_rows":
        336,

    "scenario_number_range":
        [1, 336],

    "scenario_execution_order":
        "ascending scenario_number",

    "scenario_manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "smoke_scenario_not_counted_as_full_run":
        True,

    "full_run_includes_scenario_1_again":
        True,

    "corruption_operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,

    "clean_bridge_reconstruction_contract_sha256":
        FD003_CLEAN_BRIDGE_RECONSTRUCTION_CONTRACT_SHA256_629AI,

    "trained_model_state_sha256":
        FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH,

    "per_scenario_pipeline":
        [
            "prepare authoritative clean DEV raw frame",
            "apply exactly one frozen manifest corruption",
            "normalize 18 frozen features with FIT-only mean/std",
            "rebuild 30-cycle stride-1 windows",
            "verify 4706 window unit/cycle keys unchanged",
            "deterministic eval-mode prediction",
            "canonical 50-pass MC-dropout",
            "use MC std ddof=0 as base uncertainty",
            "phase adjustment from deterministic prediction + MC std",
            "G1 clean-DEV right-ECDF transform",
            "G2 frozen engine-balanced weighted-midrank transform",
            "store label-free scenario evidence",
        ],

    "deterministic_inference_batch_size":
        512,

    "mc_passes":
        50,

    "mc_batch_size":
        512,

    "mc_seed":
        42,

    "mc_std_ddof":
        0,

    "mc_function":
        "mc_dropout_predict_fd003_629m",

    "phase_source":
        "deterministic_prediction",

    "base_uncertainty":
        "MC_dropout_std",

    "phase_adjustment_function":
        "apply_fd003_phase_adjustment_629s",

    "phase_edges_sha256":
        FD003_PHASE_EDGES_SHA256_629R,

    "phase_reference_sha256":
        {
            str(k):
                str(v)
            for k, v
            in FD003_PHASE_REFERENCE_SHA256_629R.items()
        },

    "g1_function":
        "apply_fd003_g1_right_ecdf_629s",

    "g1_reference_sha256":
        FD003_G1_REFERENCE_SHA256_629S,

    "g2_function":
        "apply_fd003_g2_weighted_midrank_629v",

    "g2_unique_values_sha256":
        FD003_G2_UNIQUE_VALUES_SHA256_629V,

    "g2_mid_mass_sha256":
        FD003_G2_MID_MASS_SHA256_629V,

    "clean_final_risk_contract_sha256":
        FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V,

    "clean_bridge_certification":
        {
            "phase_adjusted_exact":
                True,

            "phase_id_returned_and_exact":
                bool(
                    phase_id_helper_match_629am
                ),

            "g1_exact":
                True,

            "g2_final_risk_exact":
                True,
        },

    "per_scenario_retained_arrays":
        [
            "deterministic_prediction_4706",
            "mc_std_4706",
            "final_selective_risk_4706",
        ],

    "mc_matrix_retention":
        "discard after mean/std derivation; "
        "do not retain 50x4706 matrix per scenario",

    "per_scenario_summary_metadata":
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "prediction_sha256",
            "mc_std_sha256",
            "final_risk_sha256",
            "model_state_unchanged",
            "model_mode_restored",
        ],

    "dev_labels_used_during_full_execution":
        False,

    "dev_errors_scored_during_full_execution":
        False,

    "threshold_selected_during_full_execution":
        False,

    "policy_selected_during_full_execution":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "post_execution_next_stage":
        "separate DEV-only corruption outcome/risk analysis "
        "under a prospectively frozen scoring protocol",
}


FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM = hashlib.sha256(
    json.dumps(
        FD003_CORRUPTION_EXECUTION_PROTOCOL_629AM,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


# =============================================================================
# 11) Freeze clean risk-bridge certification evidence
# =============================================================================

FD003_CORRUPTION_RISK_BRIDGE_CERTIFICATION_629AM = {

    "phase_adjusted_sha256":
        _raw_sha256_629am(
            phase_adjusted_rebuilt_629am
        ),

    "expected_phase_adjusted_sha256":
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S,

    "phase_id_helper_match":
        bool(
            phase_id_helper_match_629am
        ),

    "g1_sha256":
        _raw_sha256_629am(
            g1_rebuilt_629am
        ),

    "expected_g1_sha256":
        FD003_CLEAN_DEV_G1_RISK_SHA256_629S,

    "final_risk_sha256":
        _raw_sha256_629am(
            final_risk_rebuilt_629am
        ),

    "expected_final_risk_sha256":
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,

    "all_label_free_bridge_outputs_exact":
        True,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_CORRUPTION_RISK_BRIDGE_CERTIFICATION_SHA256_629AM = hashlib.sha256(
    json.dumps(
        FD003_CORRUPTION_RISK_BRIDGE_CERTIFICATION_629AM,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


# =============================================================================
# 12) Lock state
# =============================================================================

FD003_CORRUPTION_RISK_BRIDGE_LOCKED_629AM = True
FD003_CORRUPTION_EXECUTION_PROTOCOL_LOCKED_629AM = True

FD003_ADDITIONAL_CORRUPTION_EXECUTED_629AM = False
FD003_MODEL_INFERENCE_PERFORMED_629AM = False
FD003_MC_EXECUTION_PERFORMED_629AM = False
FD003_DEV_SCORING_PERFORMED_629AM = False
FD003_THRESHOLD_SELECTED_629AM = False
FD003_POLICY_SELECTED_629AM = False
FD003_TEST_CONTENTS_READ_629AM = False
FD003_RUL_CONTENTS_READ_629AM = False

all_checks_passed_629am = True


# =============================================================================
# 13) Output
# =============================================================================

print("=" * 154)

print(
    "STEP 6.29AM — FD003 LABEL-FREE RISK-BRIDGE CERTIFICATION "
    "+ FULL CORRUPTION EXECUTION-PROTOCOL FREEZE"
)

print("=" * 154)

print("Additional corruption executed :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("DEV labels/errors used         :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)


print(
    "\n--- CLEAN LABEL-FREE BRIDGE CERTIFICATION ---"
)

print(
    "Phase-adjusted uncertainty exact:",
    (
        _raw_sha256_629am(
            phase_adjusted_rebuilt_629am
        )
        ==
        FD003_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629S
    ),
)

print(
    "Phase-ID helper output exact    :",
    phase_id_helper_match_629am,
)

print(
    "G1 risk exact                   :",
    (
        _raw_sha256_629am(
            g1_rebuilt_629am
        )
        ==
        FD003_CLEAN_DEV_G1_RISK_SHA256_629S
    ),
)

print(
    "Final G2 selective risk exact   :",
    (
        _raw_sha256_629am(
            final_risk_rebuilt_629am
        )
        ==
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V
    ),
)


print(
    "\n--- CANONICAL FULL-RUN PARAMETERS ---"
)

print(
    "Manifest scenarios              :",
    336,
)

print(
    "Execution order                 : "
    "ascending scenario_number 1..336"
)

print(
    "Deterministic batch size        :",
    512,
)

print(
    "MC passes / batch / seed / ddof :",
    (
        50,
        512,
        42,
        0,
    ),
)

print(
    "DEV labels during execution     :",
    False,
)

print(
    "MC matrices retained/scenario  :",
    False,
)

print(
    "Retained/scenario               : "
    "deterministic prediction, MC std, final risk"
)


print(
    "\n--- RISK-BRIDGE FUNCTION AUDIT ---"
)

for name_629am, audit_629am in (
    FD003_CORRUPTION_RISK_BRIDGE_FUNCTION_AUDIT_629AM.items()
):

    print(
        f"{name_629am:40s}:",
        audit_629am[
            "semantic_sha256"
        ],
    )


print(
    "\nRisk-bridge certification SHA-256 :",
    FD003_CORRUPTION_RISK_BRIDGE_CERTIFICATION_SHA256_629AM,
)

print(
    "Execution-protocol SHA-256        :",
    FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
)

print(
    "\nPASS — the clean frozen MC→phase→G1→G2 label-free risk chain "
    "was reproduced exactly without new inference/MC execution, and the "
    "full 336-scenario corruption-execution protocol is now frozen."
)

print(
    "Next: execute the frozen 336 scenarios label-free only. "
    "DEV targets/errors will remain unused until a separate scoring protocol "
    "is frozen after execution."
)

In [ ]:
# STEP 6.29AN
# FD003 FULL-RUN RELIABILITY-HELPER RETURN-SEMANTICS LOCK
#
# Source/bytecode inspection ONLY.
# Does NOT call any reliability helper.
#
# NO corruption.
# NO inference.
# NO MC execution.
# NO DEV labels/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.

import ast
import hashlib
import inspect
import json
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629am", False)):
    raise RuntimeError(
        "STEP 6.29AM must pass before STEP 6.29AN."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AM", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AN."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AM", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AN."
    )

required_629an = [
    "FD003_CORRUPTION_RISK_BRIDGE_FUNCTION_AUDIT_629AM",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "mc_dropout_predict_fd003_629m",
    "apply_fd003_phase_adjustment_629s",
    "apply_fd003_g1_right_ecdf_629s",
    "apply_fd003_g2_weighted_midrank_629v",
]

missing_629an = [
    name
    for name in required_629an
    if name not in globals()
]

if missing_629an:
    raise RuntimeError(
        "Missing required STEP 6.29AN object(s): "
        + ", ".join(missing_629an)
    )


# =============================================================================
# 2) Fingerprint helper identical to STEP 6.29AM convention
# =============================================================================

def _semantic_fingerprint_629an(fn):

    signature_629an = str(
        inspect.signature(fn)
    )

    code_629an = fn.__code__

    bytecode_sha_629an = hashlib.sha256(
        code_629an.co_code
    ).hexdigest()

    constants_sha_629an = hashlib.sha256(
        repr(
            code_629an.co_consts
        ).encode("utf-8")
    ).hexdigest()

    names_sha_629an = hashlib.sha256(
        repr(
            code_629an.co_names
        ).encode("utf-8")
    ).hexdigest()

    semantic_sha_629an = hashlib.sha256(
        json.dumps(
            {
                "signature":
                    signature_629an,

                "bytecode_sha256":
                    bytecode_sha_629an,

                "constants_sha256":
                    constants_sha_629an,

                "names_sha256":
                    names_sha_629an,
            },
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    return (
        signature_629an,
        semantic_sha_629an,
    )


# =============================================================================
# 3) Recover return expressions without executing helpers
# =============================================================================

helper_map_629an = {
    "mc_dropout_predict_fd003_629m":
        mc_dropout_predict_fd003_629m,

    "apply_fd003_phase_adjustment_629s":
        apply_fd003_phase_adjustment_629s,

    "apply_fd003_g1_right_ecdf_629s":
        apply_fd003_g1_right_ecdf_629s,

    "apply_fd003_g2_weighted_midrank_629v":
        apply_fd003_g2_weighted_midrank_629v,
}


rows_629an = []
sources_629an = {}


for name_629an, fn_629an in helper_map_629an.items():

    signature_629an, semantic_sha_629an = (
        _semantic_fingerprint_629an(
            fn_629an
        )
    )

    expected_semantic_sha_629an = (
        FD003_CORRUPTION_RISK_BRIDGE_FUNCTION_AUDIT_629AM[
            name_629an
        ][
            "semantic_sha256"
        ]
    )

    fingerprint_exact_629an = bool(
        semantic_sha_629an
        ==
        expected_semantic_sha_629an
    )

    if not fingerprint_exact_629an:
        raise RuntimeError(
            f"Frozen helper fingerprint changed: {name_629an}"
        )

    try:
        source_629an = inspect.getsource(
            fn_629an
        )
    except Exception as exc:
        raise RuntimeError(
            f"Source unavailable for {name_629an}."
        ) from exc

    sources_629an[
        name_629an
    ] = source_629an

    source_sha_629an = hashlib.sha256(
        source_629an.encode("utf-8")
    ).hexdigest()

    tree_629an = ast.parse(
        source_629an
    )

    return_exprs_629an = []

    for node_629an in ast.walk(
        tree_629an
    ):

        if isinstance(
            node_629an,
            ast.Return,
        ):

            if node_629an.value is None:

                return_exprs_629an.append(
                    "None"
                )

            else:

                return_exprs_629an.append(
                    ast.unparse(
                        node_629an.value
                    )
                )

    rows_629an.append(
        {
            "name":
                name_629an,

            "signature":
                signature_629an,

            "semantic_fingerprint_exact":
                fingerprint_exact_629an,

            "source_sha256":
                source_sha_629an,

            "return_count":
                int(
                    len(
                        return_exprs_629an
                    )
                ),

            "return_expressions":
                return_exprs_629an,
        }
    )


FD003_FULL_RUN_HELPER_RETURN_AUDIT_629AN = pd.DataFrame(
    rows_629an
)


# =============================================================================
# 4) Freeze audit contract
# =============================================================================

return_contract_629an = {

    row["name"]: {
        "signature":
            row["signature"],

        "semantic_fingerprint_exact":
            bool(
                row[
                    "semantic_fingerprint_exact"
                ]
            ),

        "source_sha256":
            row[
                "source_sha256"
            ],

        "return_count":
            int(
                row[
                    "return_count"
                ]
            ),

        "return_expressions":
            list(
                row[
                    "return_expressions"
                ]
            ),
    }

    for row in rows_629an
}


FD003_FULL_RUN_HELPER_RETURN_CONTRACT_629AN = {

    "step":
        "6.29AN",

    "dataset":
        "FD003",

    "execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,

    "helper_semantics":
        return_contract_629an,

    "helpers_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_labels_used":
        False,

    "dev_error_scoring_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN = hashlib.sha256(
    json.dumps(
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_629AN,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


FD003_FULL_RUN_HELPER_RETURN_LOCKED_629AN = True

FD003_ADDITIONAL_CORRUPTION_EXECUTED_629AN = False
FD003_MODEL_INFERENCE_PERFORMED_629AN = False
FD003_MC_EXECUTION_PERFORMED_629AN = False
FD003_DEV_SCORING_PERFORMED_629AN = False
FD003_THRESHOLD_SELECTED_629AN = False
FD003_POLICY_SELECTED_629AN = False
FD003_TEST_CONTENTS_READ_629AN = False
FD003_RUL_CONTENTS_READ_629AN = False

all_checks_passed_629an = True


# =============================================================================
# 5) Output
# =============================================================================

print("=" * 154)

print(
    "STEP 6.29AN — FD003 FULL-RUN RELIABILITY-HELPER "
    "RETURN-SEMANTICS LOCK"
)

print("=" * 154)

print("Helpers executed              :", False)
print("Additional corruption executed:", False)
print("Model inference performed     :", False)
print("MC-dropout execution          :", False)
print("DEV labels/errors used        :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print(
    "\n--- EXACT HELPER RETURN SEMANTICS ---"
)

for row_629an in rows_629an:

    print(
        "\n"
        + "-" * 118
    )

    print(
        row_629an[
            "name"
        ]
    )

    print(
        "Signature                    :",
        row_629an[
            "signature"
        ],
    )

    print(
        "Frozen fingerprint exact     :",
        row_629an[
            "semantic_fingerprint_exact"
        ],
    )

    print(
        "Source SHA-256               :",
        row_629an[
            "source_sha256"
        ],
    )

    print(
        "Return expression(s)         :",
        row_629an[
            "return_expressions"
        ],
    )

print(
    "\nHelper-return contract SHA-256:",
    FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
)

print(
    "\nPASS — the exact return semantics of the frozen "
    "MC→phase→G1→G2 helpers are now locked without executing them."
)

print(
    "Next: execute the frozen 336-scenario label-free corruption run "
    "with no DEV target/error access."
)

In [ ]:
# STEP 6.29AO-A
# FD003 FROZEN CORRUPTION EXECUTION — BLOCK 1/5 (BIAS, scenarios 001..096)
#
# Operational batching only; scientific protocol remains STEP 6.29AM.
# LABEL-FREE: no DEV targets/errors, threshold/policy, TEST, or official RUL.

import hashlib, inspect, json, time
import numpy as np
import pandas as pd
import torch


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629an", False)):
    raise RuntimeError(
        "STEP 6.29AN must pass before STEP 6.29AO-A."
    )

required = [
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_629AM",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_629AN",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_AUTHORITATIVE_DEV_RAW_629AH",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_NORMALIZED_FEATURE_NAMES_629E",
    "FD003_FIT_FEATURE_MEANS_629E",
    "FD003_FIT_FEATURE_STDS_629E",
    "DEV_WINDOW_UNIT_IDS_FD003_629F",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "_state_dict_sha256_629k",
    "mc_dropout_predict_fd003_629m",
    "apply_fd003_phase_adjustment_629s",
    "apply_fd003_g1_right_ecdf_629s",
    "apply_fd003_g2_weighted_midrank_629v",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing STEP 6.29AO-A object(s): "
        + ", ".join(missing)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AN", True)):
    raise RuntimeError(
        "Unexpected TEST-access state."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AN", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state."
    )

if bool(
    globals().get(
        "FD003_CORRUPTION_BIAS_BLOCK_COMPLETE_629AOA",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AO-A already completed."
    )


def _sha_arr_629aoa(x):

    a = np.ascontiguousarray(
        np.asarray(x)
    )

    if a.dtype.hasobject:
        raise RuntimeError(
            "Object dtype refused for hashing."
        )

    return hashlib.sha256(
        a.tobytes(order="C")
    ).hexdigest()


def _sha_json_629aoa(x):

    return hashlib.sha256(
        json.dumps(
            x,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()


if (
    _sha_json_629aoa(
        FD003_CORRUPTION_EXECUTION_PROTOCOL_629AM
    )
    !=
    FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM
):
    raise RuntimeError(
        "Frozen execution protocol changed."
    )

if (
    _sha_json_629aoa(
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_629AN
    )
    !=
    FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN
):
    raise RuntimeError(
        "Frozen helper-return contract changed."
    )


# =============================================================================
# 2) Freeze operational block 001..096
# =============================================================================

block = (
    FD003_CORRUPTION_MANIFEST_629AF
    .query(
        "1 <= scenario_number <= 96"
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

if block.shape != (96, 9):
    raise RuntimeError(
        "Bias block shape is not (96, 9)."
    )

if not bool(
    (
        block["family"]
        ==
        "bias"
    ).all()
):
    raise RuntimeError(
        "Scenarios 001..096 are not all bias."
    )

if not np.array_equal(
    block[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        97,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Bias block ordering is not exactly 001..096."
    )


FD003_CORRUPTION_BIAS_BLOCK_PROTOCOL_629AOA = {

    "step":
        "6.29AO-A",

    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,

    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,

    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,

    "scenario_start":
        1,

    "scenario_end":
        96,

    "scenario_count":
        96,

    "family":
        "bias",

    "execution_order":
        "ascending scenario_number",

    "dev_labels_used":
        False,

    "dev_errors_scored":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_CORRUPTION_BIAS_BLOCK_PROTOCOL_SHA256_629AOA = (
    _sha_json_629aoa(
        FD003_CORRUPTION_BIAS_BLOCK_PROTOCOL_629AOA
    )
)


# =============================================================================
# 3) Generic frozen execution engine
# =============================================================================

def _run_fd003_block_629aoa(
    block_df
):

    raw_features = [
        str(x)
        for x in FD003_SELECTED_RAW_FEATURES_629E
    ]

    norm_features = [
        str(x)
        for x in FD003_NORMALIZED_FEATURE_NAMES_629E
    ]

    means = np.asarray(
        FD003_FIT_FEATURE_MEANS_629E,
        dtype=np.float64,
    ).reshape(
        1,
        -1,
    )

    stds = np.asarray(
        FD003_FIT_FEATURE_STDS_629E,
        dtype=np.float64,
    ).reshape(
        1,
        -1,
    )

    clean_units = np.asarray(
        DEV_WINDOW_UNIT_IDS_FD003_629F,
        dtype=np.int64,
    ).reshape(-1)

    clean_cycles = np.asarray(
        DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
        dtype=np.int64,
    ).reshape(-1)

    base = FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
        "base_frame_preparation"
    ](
        FD003_AUTHORITATIVE_DEV_RAW_629AH
    )

    if base.shape != (5286, 27):
        raise RuntimeError(
            "Prepared DEV corruption base shape changed."
        )

    model = FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH

    device = next(
        model.parameters()
    ).device

    model_sha_start = _state_dict_sha256_629k(
        model
    )

    mode_start = bool(
        model.training
    )

    if (
        model_sha_start
        !=
        FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
    ):
        raise RuntimeError(
            "Model state changed before block execution."
        )

    n = len(
        block_df
    )

    preds = np.empty(
        (
            n,
            4706,
        ),
        dtype=np.float64,
    )

    mcstds = np.empty(
        (
            n,
            4706,
        ),
        dtype=np.float64,
    )

    risks = np.empty(
        (
            n,
            4706,
        ),
        dtype=np.float64,
    )

    summary = []

    t0 = time.perf_counter()

    try:

        for i, (
            _,
            row,
        ) in enumerate(
            block_df.iterrows()
        ):

            sn = int(
                row[
                    "scenario_number"
                ]
            )

            sid = str(
                row[
                    "scenario_id"
                ]
            )

            sensor = str(
                row[
                    "sensor"
                ]
            )

            corrupted = FD003_CORRUPTION_OPERATOR_CONTRACT_629AJ[
                "apply_manifest_scenario"
            ](
                base,
                row,
            )

            if (
                corrupted.shape
                !=
                base.shape
                or
                list(
                    corrupted.columns
                )
                !=
                list(
                    base.columns
                )
            ):
                raise RuntimeError(
                    f"{sid}: corruption changed frame structure."
                )

            if not corrupted.index.equals(
                base.index
            ):
                raise RuntimeError(
                    f"{sid}: corruption changed frame index."
                )

            non_target = [
                c
                for c in base.columns
                if c != sensor
            ]

            if not base[
                non_target
            ].equals(
                corrupted[
                    non_target
                ]
            ):
                raise RuntimeError(
                    f"{sid}: a non-target raw column changed."
                )

            raw = corrupted[
                raw_features
            ].to_numpy(
                dtype=np.float64
            )

            if not np.isfinite(
                raw
            ).all():
                raise RuntimeError(
                    f"{sid}: non-finite corrupted model input."
                )

            z = (
                raw
                -
                means
            ) / stds

            if not np.isfinite(
                z
            ).all():
                raise RuntimeError(
                    f"{sid}: non-finite normalized input."
                )

            frame = corrupted[
                [
                    "unit_id",
                    "cycle",
                ]
            ].copy()

            for j, name in enumerate(
                norm_features
            ):

                frame[
                    name
                ] = z[
                    :,
                    j,
                ]

            wb = []
            wu = []
            wc = []

            for uid, g in frame.groupby(
                "unit_id",
                sort=False,
            ):

                g = g.sort_values(
                    "cycle",
                    kind="mergesort",
                )

                fv = g[
                    norm_features
                ].to_numpy(
                    dtype=np.float64
                )

                cyc = g[
                    "cycle"
                ].to_numpy(
                    dtype=np.int64
                )

                for start in range(
                    0,
                    len(g) - 29,
                ):

                    stop = (
                        start
                        +
                        30
                    )

                    wb.append(
                        fv[
                            start:stop,
                            :,
                        ]
                    )

                    wu.append(
                        int(uid)
                    )

                    wc.append(
                        int(
                            cyc[
                                stop - 1
                            ]
                        )
                    )

            X = np.asarray(
                wb,
                dtype=np.float32,
            )

            wu = np.asarray(
                wu,
                dtype=np.int64,
            )

            wc = np.asarray(
                wc,
                dtype=np.int64,
            )

            if X.shape != (4706, 30, 18):
                raise RuntimeError(
                    f"{sid}: window shape changed."
                )

            if not np.array_equal(
                wu,
                clean_units,
            ):
                raise RuntimeError(
                    f"{sid}: window engine membership changed."
                )

            if not np.array_equal(
                wc,
                clean_cycles,
            ):
                raise RuntimeError(
                    f"{sid}: window endpoint cycles changed."
                )

            # -------------------------------------------------------------
            # Deterministic inference
            # -------------------------------------------------------------

            model.eval()

            parts = []

            with torch.no_grad():

                for bs in range(
                    0,
                    4706,
                    512,
                ):

                    xb = torch.from_numpy(
                        X[
                            bs:bs + 512
                        ]
                    ).to(
                        device=device,
                        dtype=torch.float32,
                    )

                    parts.append(
                        model(
                            xb
                        )
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(
                            np.float64,
                            copy=False,
                        )
                        .reshape(-1)
                    )

            pred = np.concatenate(
                parts
            )

            if (
                pred.shape
                !=
                (4706,)
                or
                not np.isfinite(
                    pred
                ).all()
            ):
                raise RuntimeError(
                    f"{sid}: invalid deterministic prediction."
                )

            # -------------------------------------------------------------
            # Frozen MC helper: exact STEP 6.29AN return semantics
            # -------------------------------------------------------------

            (
                all_mc,
                mc_mean,
                mc_std,
                mc_runtime,
            ) = mc_dropout_predict_fd003_629m(
                X,
                model,
                device,
                passes=50,
                batch_size=512,
                seed=42,
            )

            if np.asarray(
                all_mc
            ).shape != (50, 4706):
                raise RuntimeError(
                    f"{sid}: MC matrix shape changed."
                )

            mc_std = np.asarray(
                mc_std,
                dtype=np.float64,
            ).reshape(-1)

            if (
                mc_std.shape
                !=
                (4706,)
                or
                not np.isfinite(
                    mc_std
                ).all()
            ):
                raise RuntimeError(
                    f"{sid}: invalid MC std."
                )

            adjusted = np.asarray(
                apply_fd003_phase_adjustment_629s(
                    pred,
                    mc_std,
                ),
                dtype=np.float64,
            ).reshape(-1)

            g1 = np.asarray(
                apply_fd003_g1_right_ecdf_629s(
                    adjusted
                ),
                dtype=np.float64,
            ).reshape(-1)

            risk = np.asarray(
                apply_fd003_g2_weighted_midrank_629v(
                    g1
                ),
                dtype=np.float64,
            ).reshape(-1)

            if (
                adjusted.shape != (4706,)
                or
                g1.shape != (4706,)
                or
                risk.shape != (4706,)
            ):
                raise RuntimeError(
                    f"{sid}: reliability-vector shape changed."
                )

            if (
                not np.isfinite(
                    risk
                ).all()
                or
                not bool(
                    (
                        (risk >= 0.0)
                        &
                        (risk <= 1.0)
                    ).all()
                )
            ):
                raise RuntimeError(
                    f"{sid}: invalid final selective risk."
                )

            if (
                _state_dict_sha256_629k(
                    model
                )
                !=
                model_sha_start
            ):
                raise RuntimeError(
                    f"{sid}: model state changed."
                )

            if bool(
                model.training
            ):
                raise RuntimeError(
                    f"{sid}: model mode was not restored to eval."
                )

            preds[
                i
            ] = pred

            mcstds[
                i
            ] = mc_std

            risks[
                i
            ] = risk

            summary.append(
                {
                    "scenario_number":
                        sn,

                    "scenario_id":
                        sid,

                    "family":
                        str(
                            row[
                                "family"
                            ]
                        ),

                    "sensor":
                        sensor,

                    "severity":
                        str(
                            row[
                                "severity"
                            ]
                        ),

                    "prediction_sha256":
                        _sha_arr_629aoa(
                            pred
                        ),

                    "mc_std_sha256":
                        _sha_arr_629aoa(
                            mc_std
                        ),

                    "final_risk_sha256":
                        _sha_arr_629aoa(
                            risk
                        ),

                    "model_state_unchanged":
                        True,

                    "model_mode_restored":
                        True,
                }
            )

            # MC matrix is deliberately not retained.
            del all_mc
            del mc_mean
            del mc_std
            del adjusted
            del g1
            del risk
            del X
            del wb

            if (
                sn % 16 == 0
                or
                sn
                ==
                int(
                    block_df[
                        "scenario_number"
                    ].iloc[-1]
                )
            ):
                print(
                    f"Completed frozen scenario {sn:03d}"
                )

    finally:

        model.train(
            mode_start
        )

    model_sha_end = _state_dict_sha256_629k(
        model
    )

    if model_sha_end != model_sha_start:
        raise RuntimeError(
            "Model state changed across the block."
        )

    if bool(
        model.training
    ) != mode_start:
        raise RuntimeError(
            "Model mode was not restored after the block."
        )

    return {
        "predictions":
            preds,

        "mc_stds":
            mcstds,

        "risks":
            risks,

        "summary":
            summary,

        "runtime_seconds":
            float(
                time.perf_counter()
                -
                t0
            ),

        "model_sha_start":
            model_sha_start,

        "model_sha_end":
            model_sha_end,
    }


# Lock execution-engine semantics for later blocks.
c = _run_fd003_block_629aoa.__code__

FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA = (
    _sha_json_629aoa(
        {
            "signature":
                str(
                    inspect.signature(
                        _run_fd003_block_629aoa
                    )
                ),

            "bytecode_sha256":
                hashlib.sha256(
                    c.co_code
                ).hexdigest(),

            "constants_sha256":
                hashlib.sha256(
                    repr(
                        c.co_consts
                    ).encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "names_sha256":
                hashlib.sha256(
                    repr(
                        c.co_names
                    ).encode(
                        "utf-8"
                    )
                ).hexdigest(),
        }
    )
)


# =============================================================================
# 4) Execute frozen bias scenarios 001..096
# =============================================================================

_result = _run_fd003_block_629aoa(
    block
)

FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA = (
    _result[
        "predictions"
    ]
)

FD003_CORRUPTION_BIAS_MC_STD_629AOA = (
    _result[
        "mc_stds"
    ]
)

FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA = (
    _result[
        "risks"
    ]
)

FD003_CORRUPTION_BIAS_SUMMARY_629AOA = pd.DataFrame(
    _result[
        "summary"
    ]
)


if (
    FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA.shape
    !=
    (96, 4706)
):
    raise RuntimeError(
        "Bias prediction matrix shape changed."
    )

if (
    FD003_CORRUPTION_BIAS_MC_STD_629AOA.shape
    !=
    (96, 4706)
):
    raise RuntimeError(
        "Bias MC-STD matrix shape changed."
    )

if (
    FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA.shape
    !=
    (96, 4706)
):
    raise RuntimeError(
        "Bias final-risk matrix shape changed."
    )

if (
    FD003_CORRUPTION_BIAS_SUMMARY_629AOA.shape
    !=
    (96, 10)
):
    raise RuntimeError(
        "Bias summary shape changed."
    )


FD003_CORRUPTION_BIAS_PREDICTIONS_SHA256_629AOA = (
    _sha_arr_629aoa(
        FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA
    )
)

FD003_CORRUPTION_BIAS_MC_STD_SHA256_629AOA = (
    _sha_arr_629aoa(
        FD003_CORRUPTION_BIAS_MC_STD_629AOA
    )
)

FD003_CORRUPTION_BIAS_FINAL_RISK_SHA256_629AOA = (
    _sha_arr_629aoa(
        FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA
    )
)

FD003_CORRUPTION_BIAS_SUMMARY_SHA256_629AOA = (
    _sha_json_629aoa(
        _result[
            "summary"
        ]
    )
)


FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_629AOA = {

    "step":
        "6.29AO-A",

    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,

    "block_protocol_sha256":
        FD003_CORRUPTION_BIAS_BLOCK_PROTOCOL_SHA256_629AOA,

    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,

    "execution_engine_sha256":
        FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,

    "scenario_start":
        1,

    "scenario_end":
        96,

    "scenario_count":
        96,

    "family":
        "bias",

    "prediction_matrix_sha256":
        FD003_CORRUPTION_BIAS_PREDICTIONS_SHA256_629AOA,

    "mc_std_matrix_sha256":
        FD003_CORRUPTION_BIAS_MC_STD_SHA256_629AOA,

    "final_risk_matrix_sha256":
        FD003_CORRUPTION_BIAS_FINAL_RISK_SHA256_629AOA,

    "summary_sha256":
        FD003_CORRUPTION_BIAS_SUMMARY_SHA256_629AOA,

    "runtime_seconds":
        float(
            _result[
                "runtime_seconds"
            ]
        ),

    "model_state_start":
        _result[
            "model_sha_start"
        ],

    "model_state_end":
        _result[
            "model_sha_end"
        ],

    "model_state_unchanged":
        True,

    "model_mode_restored":
        True,

    "dev_labels_used":
        False,

    "dev_errors_scored":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA = (
    _sha_json_629aoa(
        FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_629AOA
    )
)


for a in (
    FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA,
    FD003_CORRUPTION_BIAS_MC_STD_629AOA,
    FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA,
):
    a.setflags(
        write=False
    )


FD003_CORRUPTION_BIAS_BLOCK_COMPLETE_629AOA = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AOA = 96
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AOA = False

FD003_DEV_LABELS_USED_629AOA = False
FD003_DEV_SCORING_PERFORMED_629AOA = False
FD003_THRESHOLD_SELECTED_629AOA = False
FD003_POLICY_SELECTED_629AOA = False
FD003_TEST_CONTENTS_READ_629AOA = False
FD003_RUL_CONTENTS_READ_629AOA = False

all_checks_passed_629aoa = True


# =============================================================================
# 5) Output
# =============================================================================

print("=" * 154)

print(
    "STEP 6.29AO-A — FD003 FROZEN CORRUPTION EXECUTION "
    "BLOCK 1/5: BIAS 001..096"
)

print("=" * 154)

print("Scenarios executed             :", 96)
print("Family                         :", "bias")
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- RETAINED LABEL-FREE EVIDENCE ---"
)

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA.shape,
)

print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_BIAS_MC_STD_629AOA.shape,
)

print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA.shape,
)

print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_BIAS_PREDICTIONS_SHA256_629AOA,
)

print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_BIAS_MC_STD_SHA256_629AOA,
)

print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_BIAS_FINAL_RISK_SHA256_629AOA,
)

print(
    "Scenario summary SHA-256       :",
    FD003_CORRUPTION_BIAS_SUMMARY_SHA256_629AOA,
)

print(
    "\n--- EXECUTION SAFETY ---"
)

print(
    "Model state unchanged          :",
    True,
)

print(
    "Model mode restored            :",
    True,
)

print(
    "Execution-engine SHA-256       :",
    FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,
)

print(
    "Bias-block evidence SHA-256    :",
    FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,
)

print(
    "\nPASS — frozen scenarios 001..096 were executed in ascending order "
    "through corruption→deterministic prediction→MC uncertainty→phase→G1→G2 "
    "with no DEV outcome access."
)

print(
    "Full 336-scenario execution is not yet complete. "
    "Next: inspect this block, then execute frozen noise scenarios "
    "097..144 with the same locked execution engine."
)

In [ ]:
# STEP 6.29AO-B
# FD003 FROZEN CORRUPTION EXECUTION — BLOCK 2/5 (NOISE, scenarios 097..144)
#
# Reuses the exact locked execution engine created in STEP 6.29AO-A.
# LABEL-FREE: no DEV targets/errors, threshold/policy, TEST, or official RUL.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd

# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aoa", False)):
    raise RuntimeError("STEP 6.29AO-A must pass before STEP 6.29AO-B.")

required_629aob = [
    "_run_fd003_block_629aoa",
    "FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA",
]

missing_629aob = [name for name in required_629aob if name not in globals()]
if missing_629aob:
    raise RuntimeError(
        "Missing STEP 6.29AO-B object(s): " + ", ".join(missing_629aob)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AOA", True)):
    raise RuntimeError("Unexpected TEST-access state before STEP 6.29AO-B.")

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AOA", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AO-B."
    )

if bool(globals().get("FD003_CORRUPTION_NOISE_BLOCK_COMPLETE_629AOB", False)):
    raise RuntimeError("STEP 6.29AO-B has already been completed.")

# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629aob(x):
    arr = np.ascontiguousarray(np.asarray(x))
    if arr.dtype.hasobject:
        raise RuntimeError("Object dtype refused for hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_629aob(x):
    return hashlib.sha256(
        json.dumps(
            x,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

# =============================================================================
# 3) Re-certify the already-locked execution engine
# =============================================================================

engine_code_629aob = _run_fd003_block_629aoa.__code__

engine_sha_now_629aob = _sha_json_629aob(
    {
        "signature": str(inspect.signature(_run_fd003_block_629aoa)),
        "bytecode_sha256": hashlib.sha256(
            engine_code_629aob.co_code
        ).hexdigest(),
        "constants_sha256": hashlib.sha256(
            repr(engine_code_629aob.co_consts).encode("utf-8")
        ).hexdigest(),
        "names_sha256": hashlib.sha256(
            repr(engine_code_629aob.co_names).encode("utf-8")
        ).hexdigest(),
    }
)

if engine_sha_now_629aob != FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA:
    raise RuntimeError(
        "Frozen corruption execution engine changed before noise block."
    )

# =============================================================================
# 4) Freeze operational block 097..144
# =============================================================================

block_629aob = (
    FD003_CORRUPTION_MANIFEST_629AF
    .query("97 <= scenario_number <= 144")
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

if block_629aob.shape != (48, 9):
    raise RuntimeError("Noise block shape is not (48, 9).")

if not bool((block_629aob["family"] == "noise").all()):
    raise RuntimeError("Scenarios 097..144 are not all noise.")

if not np.array_equal(
    block_629aob["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(97, 145, dtype=np.int64),
):
    raise RuntimeError("Noise block ordering is not exactly 097..144.")

FD003_CORRUPTION_NOISE_BLOCK_PROTOCOL_629AOB = {
    "step": "6.29AO-B",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,
    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
    "execution_engine_sha256":
        engine_sha_now_629aob,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,
    "scenario_start": 97,
    "scenario_end": 144,
    "scenario_count": 48,
    "family": "noise",
    "execution_order": "ascending scenario_number",
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_NOISE_BLOCK_PROTOCOL_SHA256_629AOB = _sha_json_629aob(
    FD003_CORRUPTION_NOISE_BLOCK_PROTOCOL_629AOB
)

# =============================================================================
# 5) Execute exact frozen noise block
# =============================================================================

result_629aob = _run_fd003_block_629aoa(block_629aob)

FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB = result_629aob["predictions"]
FD003_CORRUPTION_NOISE_MC_STD_629AOB = result_629aob["mc_stds"]
FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB = result_629aob["risks"]
FD003_CORRUPTION_NOISE_SUMMARY_629AOB = pd.DataFrame(
    result_629aob["summary"]
)

# =============================================================================
# 6) Structural audits
# =============================================================================

if FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB.shape != (48, 4706):
    raise RuntimeError("Noise prediction matrix shape changed.")

if FD003_CORRUPTION_NOISE_MC_STD_629AOB.shape != (48, 4706):
    raise RuntimeError("Noise MC-STD matrix shape changed.")

if FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB.shape != (48, 4706):
    raise RuntimeError("Noise final-risk matrix shape changed.")

if FD003_CORRUPTION_NOISE_SUMMARY_629AOB.shape != (48, 10):
    raise RuntimeError("Noise summary shape changed.")

# =============================================================================
# 7) Fingerprints
# =============================================================================

FD003_CORRUPTION_NOISE_PREDICTIONS_SHA256_629AOB = _sha_arr_629aob(
    FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB
)

FD003_CORRUPTION_NOISE_MC_STD_SHA256_629AOB = _sha_arr_629aob(
    FD003_CORRUPTION_NOISE_MC_STD_629AOB
)

FD003_CORRUPTION_NOISE_FINAL_RISK_SHA256_629AOB = _sha_arr_629aob(
    FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB
)

FD003_CORRUPTION_NOISE_SUMMARY_SHA256_629AOB = _sha_json_629aob(
    result_629aob["summary"]
)

# =============================================================================
# 8) Evidence lock
# =============================================================================

FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_629AOB = {
    "step": "6.29AO-B",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "block_protocol_sha256":
        FD003_CORRUPTION_NOISE_BLOCK_PROTOCOL_SHA256_629AOB,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "execution_engine_sha256":
        engine_sha_now_629aob,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,
    "scenario_start": 97,
    "scenario_end": 144,
    "scenario_count": 48,
    "cumulative_scenarios_executed": 144,
    "family": "noise",
    "prediction_matrix_sha256":
        FD003_CORRUPTION_NOISE_PREDICTIONS_SHA256_629AOB,
    "mc_std_matrix_sha256":
        FD003_CORRUPTION_NOISE_MC_STD_SHA256_629AOB,
    "final_risk_matrix_sha256":
        FD003_CORRUPTION_NOISE_FINAL_RISK_SHA256_629AOB,
    "summary_sha256":
        FD003_CORRUPTION_NOISE_SUMMARY_SHA256_629AOB,
    "runtime_seconds":
        float(result_629aob["runtime_seconds"]),
    "model_state_start":
        result_629aob["model_sha_start"],
    "model_state_end":
        result_629aob["model_sha_end"],
    "model_state_unchanged": True,
    "model_mode_restored": True,
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB = _sha_json_629aob(
    FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_629AOB
)

for arr_629aob in (
    FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB,
    FD003_CORRUPTION_NOISE_MC_STD_629AOB,
    FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB,
):
    arr_629aob.setflags(write=False)

FD003_CORRUPTION_NOISE_BLOCK_COMPLETE_629AOB = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AOB = 144
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AOB = False

FD003_DEV_LABELS_USED_629AOB = False
FD003_DEV_SCORING_PERFORMED_629AOB = False
FD003_THRESHOLD_SELECTED_629AOB = False
FD003_POLICY_SELECTED_629AOB = False
FD003_TEST_CONTENTS_READ_629AOB = False
FD003_RUL_CONTENTS_READ_629AOB = False

all_checks_passed_629aob = True

# =============================================================================
# 9) Output
# =============================================================================

print("=" * 154)
print(
    "STEP 6.29AO-B — FD003 FROZEN CORRUPTION EXECUTION "
    "BLOCK 2/5: NOISE 097..144"
)
print("=" * 154)

print("Scenarios executed this block  :", 48)
print("Cumulative scenarios executed  :", 144)
print("Family                         :", "noise")
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- RETAINED LABEL-FREE EVIDENCE ---")

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB.shape,
)
print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_NOISE_MC_STD_629AOB.shape,
)
print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB.shape,
)
print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_NOISE_PREDICTIONS_SHA256_629AOB,
)
print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_NOISE_MC_STD_SHA256_629AOB,
)
print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_NOISE_FINAL_RISK_SHA256_629AOB,
)
print(
    "Scenario summary SHA-256       :",
    FD003_CORRUPTION_NOISE_SUMMARY_SHA256_629AOB,
)

print("\n--- EXECUTION SAFETY ---")

print(
    "Execution engine unchanged     :",
    engine_sha_now_629aob
    == FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,
)
print("Model state unchanged          :", True)
print("Model mode restored            :", True)
print(
    "Noise-block evidence SHA-256   :",
    FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,
)

print(
    "\nPASS — frozen scenarios 097..144 were executed using the exact "
    "same locked label-free execution engine."
)
print(
    "Full run progress: 144 / 336 scenarios. "
    "Next: execute drift scenarios 145..240 only after inspecting this output."
)

In [ ]:
# STEP 6.29AO-C
# FD003 FROZEN CORRUPTION EXECUTION — BLOCK 3/5 (DRIFT, scenarios 145..240)
#
# Reuses the exact locked execution engine created in STEP 6.29AO-A.
# LABEL-FREE: no DEV targets/errors, threshold/policy, TEST, or official RUL.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd

# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aob", False)):
    raise RuntimeError("STEP 6.29AO-B must pass before STEP 6.29AO-C.")

required_629aoc = [
    "_run_fd003_block_629aoa",
    "FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB",
]

missing_629aoc = [
    name for name in required_629aoc
    if name not in globals()
]

if missing_629aoc:
    raise RuntimeError(
        "Missing STEP 6.29AO-C object(s): " + ", ".join(missing_629aoc)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AOB", True)):
    raise RuntimeError("Unexpected TEST-access state before STEP 6.29AO-C.")

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AOB", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AO-C."
    )

if bool(globals().get("FD003_CORRUPTION_DRIFT_BLOCK_COMPLETE_629AOC", False)):
    raise RuntimeError("STEP 6.29AO-C has already been completed.")

# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629aoc(x):
    arr = np.ascontiguousarray(np.asarray(x))
    if arr.dtype.hasobject:
        raise RuntimeError("Object dtype refused for hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_629aoc(x):
    return hashlib.sha256(
        json.dumps(
            x,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

# =============================================================================
# 3) Re-certify the already-locked execution engine
# =============================================================================

engine_code_629aoc = _run_fd003_block_629aoa.__code__

engine_sha_now_629aoc = _sha_json_629aoc(
    {
        "signature": str(inspect.signature(_run_fd003_block_629aoa)),
        "bytecode_sha256": hashlib.sha256(
            engine_code_629aoc.co_code
        ).hexdigest(),
        "constants_sha256": hashlib.sha256(
            repr(engine_code_629aoc.co_consts).encode("utf-8")
        ).hexdigest(),
        "names_sha256": hashlib.sha256(
            repr(engine_code_629aoc.co_names).encode("utf-8")
        ).hexdigest(),
    }
)

if engine_sha_now_629aoc != FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA:
    raise RuntimeError(
        "Frozen corruption execution engine changed before drift block."
    )

# =============================================================================
# 4) Freeze operational block 145..240
# =============================================================================

block_629aoc = (
    FD003_CORRUPTION_MANIFEST_629AF
    .query("145 <= scenario_number <= 240")
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

if block_629aoc.shape != (96, 9):
    raise RuntimeError("Drift block shape is not (96, 9).")

if not bool((block_629aoc["family"] == "drift").all()):
    raise RuntimeError("Scenarios 145..240 are not all drift.")

if not np.array_equal(
    block_629aoc["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(145, 241, dtype=np.int64),
):
    raise RuntimeError("Drift block ordering is not exactly 145..240.")

FD003_CORRUPTION_DRIFT_BLOCK_PROTOCOL_629AOC = {
    "step": "6.29AO-C",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,
    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
    "execution_engine_sha256":
        engine_sha_now_629aoc,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,
    "scenario_start": 145,
    "scenario_end": 240,
    "scenario_count": 96,
    "family": "drift",
    "execution_order": "ascending scenario_number",
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_DRIFT_BLOCK_PROTOCOL_SHA256_629AOC = _sha_json_629aoc(
    FD003_CORRUPTION_DRIFT_BLOCK_PROTOCOL_629AOC
)

# =============================================================================
# 5) Execute exact frozen drift block
# =============================================================================

result_629aoc = _run_fd003_block_629aoa(block_629aoc)

FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC = result_629aoc["predictions"]
FD003_CORRUPTION_DRIFT_MC_STD_629AOC = result_629aoc["mc_stds"]
FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC = result_629aoc["risks"]
FD003_CORRUPTION_DRIFT_SUMMARY_629AOC = pd.DataFrame(
    result_629aoc["summary"]
)

# =============================================================================
# 6) Structural audits
# =============================================================================

if FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC.shape != (96, 4706):
    raise RuntimeError("Drift prediction matrix shape changed.")

if FD003_CORRUPTION_DRIFT_MC_STD_629AOC.shape != (96, 4706):
    raise RuntimeError("Drift MC-STD matrix shape changed.")

if FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC.shape != (96, 4706):
    raise RuntimeError("Drift final-risk matrix shape changed.")

if FD003_CORRUPTION_DRIFT_SUMMARY_629AOC.shape != (96, 10):
    raise RuntimeError("Drift summary shape changed.")

# =============================================================================
# 7) Fingerprints
# =============================================================================

FD003_CORRUPTION_DRIFT_PREDICTIONS_SHA256_629AOC = _sha_arr_629aoc(
    FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC
)

FD003_CORRUPTION_DRIFT_MC_STD_SHA256_629AOC = _sha_arr_629aoc(
    FD003_CORRUPTION_DRIFT_MC_STD_629AOC
)

FD003_CORRUPTION_DRIFT_FINAL_RISK_SHA256_629AOC = _sha_arr_629aoc(
    FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC
)

FD003_CORRUPTION_DRIFT_SUMMARY_SHA256_629AOC = _sha_json_629aoc(
    result_629aoc["summary"]
)

# =============================================================================
# 8) Evidence lock
# =============================================================================

FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_629AOC = {
    "step": "6.29AO-C",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "block_protocol_sha256":
        FD003_CORRUPTION_DRIFT_BLOCK_PROTOCOL_SHA256_629AOC,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "execution_engine_sha256":
        engine_sha_now_629aoc,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,
    "scenario_start": 145,
    "scenario_end": 240,
    "scenario_count": 96,
    "cumulative_scenarios_executed": 240,
    "family": "drift",
    "prediction_matrix_sha256":
        FD003_CORRUPTION_DRIFT_PREDICTIONS_SHA256_629AOC,
    "mc_std_matrix_sha256":
        FD003_CORRUPTION_DRIFT_MC_STD_SHA256_629AOC,
    "final_risk_matrix_sha256":
        FD003_CORRUPTION_DRIFT_FINAL_RISK_SHA256_629AOC,
    "summary_sha256":
        FD003_CORRUPTION_DRIFT_SUMMARY_SHA256_629AOC,
    "runtime_seconds":
        float(result_629aoc["runtime_seconds"]),
    "model_state_start":
        result_629aoc["model_sha_start"],
    "model_state_end":
        result_629aoc["model_sha_end"],
    "model_state_unchanged": True,
    "model_mode_restored": True,
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC = _sha_json_629aoc(
    FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_629AOC
)

for arr_629aoc in (
    FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC,
    FD003_CORRUPTION_DRIFT_MC_STD_629AOC,
    FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC,
):
    arr_629aoc.setflags(write=False)

FD003_CORRUPTION_DRIFT_BLOCK_COMPLETE_629AOC = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AOC = 240
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AOC = False

FD003_DEV_LABELS_USED_629AOC = False
FD003_DEV_SCORING_PERFORMED_629AOC = False
FD003_THRESHOLD_SELECTED_629AOC = False
FD003_POLICY_SELECTED_629AOC = False
FD003_TEST_CONTENTS_READ_629AOC = False
FD003_RUL_CONTENTS_READ_629AOC = False

all_checks_passed_629aoc = True

# =============================================================================
# 9) Output
# =============================================================================

print("=" * 154)
print(
    "STEP 6.29AO-C — FD003 FROZEN CORRUPTION EXECUTION "
    "BLOCK 3/5: DRIFT 145..240"
)
print("=" * 154)

print("Scenarios executed this block  :", 96)
print("Cumulative scenarios executed  :", 240)
print("Family                         :", "drift")
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- RETAINED LABEL-FREE EVIDENCE ---")

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC.shape,
)
print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_DRIFT_MC_STD_629AOC.shape,
)
print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC.shape,
)
print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_DRIFT_PREDICTIONS_SHA256_629AOC,
)
print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_DRIFT_MC_STD_SHA256_629AOC,
)
print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_DRIFT_FINAL_RISK_SHA256_629AOC,
)
print(
    "Scenario summary SHA-256       :",
    FD003_CORRUPTION_DRIFT_SUMMARY_SHA256_629AOC,
)

print("\n--- EXECUTION SAFETY ---")

print(
    "Execution engine unchanged     :",
    engine_sha_now_629aoc
    == FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,
)
print("Model state unchanged          :", True)
print("Model mode restored            :", True)

print(
    "Drift-block evidence SHA-256   :",
    FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,
)

print(
    "\nPASS — frozen scenarios 145..240 were executed using the exact "
    "same locked label-free execution engine."
)

print(
    "Full run progress: 240 / 336 scenarios. "
    "Next: execute dropout scenarios 241..288 only after inspecting this output."
)

In [ ]:
# STEP 6.29AO-D
# FD003 FROZEN CORRUPTION EXECUTION — BLOCK 4/5 (DROPOUT, scenarios 241..288)
#
# Reuses the exact locked execution engine created in STEP 6.29AO-A.
# LABEL-FREE: no DEV targets/errors, threshold/policy, TEST, or official RUL.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd

# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aoc", False)):
    raise RuntimeError("STEP 6.29AO-C must pass before STEP 6.29AO-D.")

required_629aod = [
    "_run_fd003_block_629aoa",
    "FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC",
]

missing_629aod = [
    name for name in required_629aod
    if name not in globals()
]

if missing_629aod:
    raise RuntimeError(
        "Missing STEP 6.29AO-D object(s): " + ", ".join(missing_629aod)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AOC", True)):
    raise RuntimeError("Unexpected TEST-access state before STEP 6.29AO-D.")

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AOC", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AO-D."
    )

if bool(globals().get("FD003_CORRUPTION_DROPOUT_BLOCK_COMPLETE_629AOD", False)):
    raise RuntimeError("STEP 6.29AO-D has already been completed.")

# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629aod(x):
    arr = np.ascontiguousarray(np.asarray(x))
    if arr.dtype.hasobject:
        raise RuntimeError("Object dtype refused for hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_629aod(x):
    return hashlib.sha256(
        json.dumps(
            x,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

# =============================================================================
# 3) Re-certify the already-locked execution engine
# =============================================================================

engine_code_629aod = _run_fd003_block_629aoa.__code__

engine_sha_now_629aod = _sha_json_629aod(
    {
        "signature": str(inspect.signature(_run_fd003_block_629aoa)),
        "bytecode_sha256": hashlib.sha256(
            engine_code_629aod.co_code
        ).hexdigest(),
        "constants_sha256": hashlib.sha256(
            repr(engine_code_629aod.co_consts).encode("utf-8")
        ).hexdigest(),
        "names_sha256": hashlib.sha256(
            repr(engine_code_629aod.co_names).encode("utf-8")
        ).hexdigest(),
    }
)

if engine_sha_now_629aod != FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA:
    raise RuntimeError(
        "Frozen corruption execution engine changed before dropout block."
    )

# =============================================================================
# 4) Freeze operational block 241..288
# =============================================================================

block_629aod = (
    FD003_CORRUPTION_MANIFEST_629AF
    .query("241 <= scenario_number <= 288")
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

if block_629aod.shape != (48, 9):
    raise RuntimeError("Dropout block shape is not (48, 9).")

if not bool((block_629aod["family"] == "dropout").all()):
    raise RuntimeError("Scenarios 241..288 are not all dropout.")

if not np.array_equal(
    block_629aod["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(241, 289, dtype=np.int64),
):
    raise RuntimeError("Dropout block ordering is not exactly 241..288.")

FD003_CORRUPTION_DROPOUT_BLOCK_PROTOCOL_629AOD = {
    "step": "6.29AO-D",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,
    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
    "execution_engine_sha256":
        engine_sha_now_629aod,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,
    "scenario_start": 241,
    "scenario_end": 288,
    "scenario_count": 48,
    "family": "dropout",
    "execution_order": "ascending scenario_number",
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_DROPOUT_BLOCK_PROTOCOL_SHA256_629AOD = _sha_json_629aod(
    FD003_CORRUPTION_DROPOUT_BLOCK_PROTOCOL_629AOD
)

# =============================================================================
# 5) Execute exact frozen dropout block
# =============================================================================

result_629aod = _run_fd003_block_629aoa(block_629aod)

FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD = result_629aod["predictions"]
FD003_CORRUPTION_DROPOUT_MC_STD_629AOD = result_629aod["mc_stds"]
FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD = result_629aod["risks"]
FD003_CORRUPTION_DROPOUT_SUMMARY_629AOD = pd.DataFrame(
    result_629aod["summary"]
)

# =============================================================================
# 6) Structural audits
# =============================================================================

if FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD.shape != (48, 4706):
    raise RuntimeError("Dropout prediction matrix shape changed.")

if FD003_CORRUPTION_DROPOUT_MC_STD_629AOD.shape != (48, 4706):
    raise RuntimeError("Dropout MC-STD matrix shape changed.")

if FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD.shape != (48, 4706):
    raise RuntimeError("Dropout final-risk matrix shape changed.")

if FD003_CORRUPTION_DROPOUT_SUMMARY_629AOD.shape != (48, 10):
    raise RuntimeError("Dropout summary shape changed.")

# =============================================================================
# 7) Fingerprints
# =============================================================================

FD003_CORRUPTION_DROPOUT_PREDICTIONS_SHA256_629AOD = _sha_arr_629aod(
    FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD
)

FD003_CORRUPTION_DROPOUT_MC_STD_SHA256_629AOD = _sha_arr_629aod(
    FD003_CORRUPTION_DROPOUT_MC_STD_629AOD
)

FD003_CORRUPTION_DROPOUT_FINAL_RISK_SHA256_629AOD = _sha_arr_629aod(
    FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD
)

FD003_CORRUPTION_DROPOUT_SUMMARY_SHA256_629AOD = _sha_json_629aod(
    result_629aod["summary"]
)

# =============================================================================
# 8) Evidence lock
# =============================================================================

FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_629AOD = {
    "step": "6.29AO-D",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "block_protocol_sha256":
        FD003_CORRUPTION_DROPOUT_BLOCK_PROTOCOL_SHA256_629AOD,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "execution_engine_sha256":
        engine_sha_now_629aod,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,
    "scenario_start": 241,
    "scenario_end": 288,
    "scenario_count": 48,
    "cumulative_scenarios_executed": 288,
    "family": "dropout",
    "prediction_matrix_sha256":
        FD003_CORRUPTION_DROPOUT_PREDICTIONS_SHA256_629AOD,
    "mc_std_matrix_sha256":
        FD003_CORRUPTION_DROPOUT_MC_STD_SHA256_629AOD,
    "final_risk_matrix_sha256":
        FD003_CORRUPTION_DROPOUT_FINAL_RISK_SHA256_629AOD,
    "summary_sha256":
        FD003_CORRUPTION_DROPOUT_SUMMARY_SHA256_629AOD,
    "runtime_seconds":
        float(result_629aod["runtime_seconds"]),
    "model_state_start":
        result_629aod["model_sha_start"],
    "model_state_end":
        result_629aod["model_sha_end"],
    "model_state_unchanged": True,
    "model_mode_restored": True,
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD = _sha_json_629aod(
    FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_629AOD
)

for arr_629aod in (
    FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD,
    FD003_CORRUPTION_DROPOUT_MC_STD_629AOD,
    FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD,
):
    arr_629aod.setflags(write=False)

FD003_CORRUPTION_DROPOUT_BLOCK_COMPLETE_629AOD = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AOD = 288
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AOD = False

FD003_DEV_LABELS_USED_629AOD = False
FD003_DEV_SCORING_PERFORMED_629AOD = False
FD003_THRESHOLD_SELECTED_629AOD = False
FD003_POLICY_SELECTED_629AOD = False
FD003_TEST_CONTENTS_READ_629AOD = False
FD003_RUL_CONTENTS_READ_629AOD = False

all_checks_passed_629aod = True

# =============================================================================
# 9) Output
# =============================================================================

print("=" * 154)
print(
    "STEP 6.29AO-D — FD003 FROZEN CORRUPTION EXECUTION "
    "BLOCK 4/5: DROPOUT 241..288"
)
print("=" * 154)

print("Scenarios executed this block  :", 48)
print("Cumulative scenarios executed  :", 288)
print("Family                         :", "dropout")
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- RETAINED LABEL-FREE EVIDENCE ---")

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD.shape,
)
print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_DROPOUT_MC_STD_629AOD.shape,
)
print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD.shape,
)
print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_DROPOUT_PREDICTIONS_SHA256_629AOD,
)
print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_DROPOUT_MC_STD_SHA256_629AOD,
)
print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_DROPOUT_FINAL_RISK_SHA256_629AOD,
)
print(
    "Scenario summary SHA-256       :",
    FD003_CORRUPTION_DROPOUT_SUMMARY_SHA256_629AOD,
)

print("\n--- EXECUTION SAFETY ---")

print(
    "Execution engine unchanged     :",
    engine_sha_now_629aod
    == FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,
)
print("Model state unchanged          :", True)
print("Model mode restored            :", True)

print(
    "Dropout-block evidence SHA-256 :",
    FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,
)

print(
    "\nPASS — frozen scenarios 241..288 were executed using the exact "
    "same locked label-free execution engine."
)

print(
    "Full run progress: 288 / 336 scenarios. "
    "Next: execute stuck scenarios 289..336 only after inspecting this output."
)

In [ ]:
# STEP 6.29AO-E
# FD003 FROZEN CORRUPTION EXECUTION — BLOCK 5/5 (STUCK, scenarios 289..336)
#
# Reuses the exact locked execution engine created in STEP 6.29AO-A.
# LABEL-FREE: no DEV targets/errors, threshold/policy, TEST, or official RUL.

import hashlib
import inspect
import json
import numpy as np
import pandas as pd

# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aod", False)):
    raise RuntimeError("STEP 6.29AO-D must pass before STEP 6.29AO-E.")

required_629aoe = [
    "_run_fd003_block_629aoa",
    "FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ",
    "FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD",
]

missing_629aoe = [
    name for name in required_629aoe
    if name not in globals()
]

if missing_629aoe:
    raise RuntimeError(
        "Missing STEP 6.29AO-E object(s): " + ", ".join(missing_629aoe)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AOD", True)):
    raise RuntimeError("Unexpected TEST-access state before STEP 6.29AO-E.")

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AOD", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AO-E."
    )

if bool(globals().get("FD003_CORRUPTION_STUCK_BLOCK_COMPLETE_629AOE", False)):
    raise RuntimeError("STEP 6.29AO-E has already been completed.")

# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629aoe(x):
    arr = np.ascontiguousarray(np.asarray(x))
    if arr.dtype.hasobject:
        raise RuntimeError("Object dtype refused for hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_629aoe(x):
    return hashlib.sha256(
        json.dumps(
            x,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

# =============================================================================
# 3) Re-certify the already-locked execution engine
# =============================================================================

engine_code_629aoe = _run_fd003_block_629aoa.__code__

engine_sha_now_629aoe = _sha_json_629aoe(
    {
        "signature": str(inspect.signature(_run_fd003_block_629aoa)),
        "bytecode_sha256": hashlib.sha256(
            engine_code_629aoe.co_code
        ).hexdigest(),
        "constants_sha256": hashlib.sha256(
            repr(engine_code_629aoe.co_consts).encode("utf-8")
        ).hexdigest(),
        "names_sha256": hashlib.sha256(
            repr(engine_code_629aoe.co_names).encode("utf-8")
        ).hexdigest(),
    }
)

if engine_sha_now_629aoe != FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA:
    raise RuntimeError(
        "Frozen corruption execution engine changed before stuck block."
    )

# =============================================================================
# 4) Freeze operational block 289..336
# =============================================================================

block_629aoe = (
    FD003_CORRUPTION_MANIFEST_629AF
    .query("289 <= scenario_number <= 336")
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

if block_629aoe.shape != (48, 9):
    raise RuntimeError("Stuck block shape is not (48, 9).")

if not bool((block_629aoe["family"] == "stuck").all()):
    raise RuntimeError("Scenarios 289..336 are not all stuck.")

if not np.array_equal(
    block_629aoe["scenario_number"].to_numpy(dtype=np.int64),
    np.arange(289, 337, dtype=np.int64),
):
    raise RuntimeError("Stuck block ordering is not exactly 289..336.")

FD003_CORRUPTION_STUCK_BLOCK_PROTOCOL_629AOE = {
    "step": "6.29AO-E",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,
    "operator_freeze_contract_sha256":
        FD003_CORRUPTION_OPERATOR_FREEZE_CONTRACT_SHA256_629AJ,
    "execution_engine_sha256":
        engine_sha_now_629aoe,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,
    "scenario_start": 289,
    "scenario_end": 336,
    "scenario_count": 48,
    "family": "stuck",
    "execution_order": "ascending scenario_number",
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_STUCK_BLOCK_PROTOCOL_SHA256_629AOE = _sha_json_629aoe(
    FD003_CORRUPTION_STUCK_BLOCK_PROTOCOL_629AOE
)

# =============================================================================
# 5) Execute exact frozen stuck block
# =============================================================================

result_629aoe = _run_fd003_block_629aoa(block_629aoe)

FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE = result_629aoe["predictions"]
FD003_CORRUPTION_STUCK_MC_STD_629AOE = result_629aoe["mc_stds"]
FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE = result_629aoe["risks"]
FD003_CORRUPTION_STUCK_SUMMARY_629AOE = pd.DataFrame(
    result_629aoe["summary"]
)

# =============================================================================
# 6) Structural audits
# =============================================================================

if FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE.shape != (48, 4706):
    raise RuntimeError("Stuck prediction matrix shape changed.")

if FD003_CORRUPTION_STUCK_MC_STD_629AOE.shape != (48, 4706):
    raise RuntimeError("Stuck MC-STD matrix shape changed.")

if FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE.shape != (48, 4706):
    raise RuntimeError("Stuck final-risk matrix shape changed.")

if FD003_CORRUPTION_STUCK_SUMMARY_629AOE.shape != (48, 10):
    raise RuntimeError("Stuck summary shape changed.")

# =============================================================================
# 7) Fingerprints
# =============================================================================

FD003_CORRUPTION_STUCK_PREDICTIONS_SHA256_629AOE = _sha_arr_629aoe(
    FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE
)

FD003_CORRUPTION_STUCK_MC_STD_SHA256_629AOE = _sha_arr_629aoe(
    FD003_CORRUPTION_STUCK_MC_STD_629AOE
)

FD003_CORRUPTION_STUCK_FINAL_RISK_SHA256_629AOE = _sha_arr_629aoe(
    FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE
)

FD003_CORRUPTION_STUCK_SUMMARY_SHA256_629AOE = _sha_json_629aoe(
    result_629aoe["summary"]
)

# =============================================================================
# 8) Evidence lock
# =============================================================================

FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_629AOE = {
    "step": "6.29AO-E",
    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,
    "block_protocol_sha256":
        FD003_CORRUPTION_STUCK_BLOCK_PROTOCOL_SHA256_629AOE,
    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,
    "execution_engine_sha256":
        engine_sha_now_629aoe,
    "previous_block_evidence_sha256":
        FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,
    "scenario_start": 289,
    "scenario_end": 336,
    "scenario_count": 48,
    "cumulative_scenarios_executed": 336,
    "family": "stuck",
    "prediction_matrix_sha256":
        FD003_CORRUPTION_STUCK_PREDICTIONS_SHA256_629AOE,
    "mc_std_matrix_sha256":
        FD003_CORRUPTION_STUCK_MC_STD_SHA256_629AOE,
    "final_risk_matrix_sha256":
        FD003_CORRUPTION_STUCK_FINAL_RISK_SHA256_629AOE,
    "summary_sha256":
        FD003_CORRUPTION_STUCK_SUMMARY_SHA256_629AOE,
    "runtime_seconds":
        float(result_629aoe["runtime_seconds"]),
    "model_state_start":
        result_629aoe["model_sha_start"],
    "model_state_end":
        result_629aoe["model_sha_end"],
    "model_state_unchanged": True,
    "model_mode_restored": True,
    "dev_labels_used": False,
    "dev_errors_scored": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_SHA256_629AOE = _sha_json_629aoe(
    FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_629AOE
)

for arr_629aoe in (
    FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE,
    FD003_CORRUPTION_STUCK_MC_STD_629AOE,
    FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE,
):
    arr_629aoe.setflags(write=False)

FD003_CORRUPTION_STUCK_BLOCK_COMPLETE_629AOE = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AOE = 336
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AOE = True

FD003_DEV_LABELS_USED_629AOE = False
FD003_DEV_SCORING_PERFORMED_629AOE = False
FD003_THRESHOLD_SELECTED_629AOE = False
FD003_POLICY_SELECTED_629AOE = False
FD003_TEST_CONTENTS_READ_629AOE = False
FD003_RUL_CONTENTS_READ_629AOE = False

all_checks_passed_629aoe = True

# =============================================================================
# 9) Output
# =============================================================================

print("=" * 154)
print(
    "STEP 6.29AO-E — FD003 FROZEN CORRUPTION EXECUTION "
    "BLOCK 5/5: STUCK 289..336"
)
print("=" * 154)

print("Scenarios executed this block  :", 48)
print("Cumulative scenarios executed  :", 336)
print("Family                         :", "stuck")
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- RETAINED LABEL-FREE EVIDENCE ---")

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE.shape,
)
print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_STUCK_MC_STD_629AOE.shape,
)
print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE.shape,
)
print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_STUCK_PREDICTIONS_SHA256_629AOE,
)
print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_STUCK_MC_STD_SHA256_629AOE,
)
print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_STUCK_FINAL_RISK_SHA256_629AOE,
)
print(
    "Scenario summary SHA-256       :",
    FD003_CORRUPTION_STUCK_SUMMARY_SHA256_629AOE,
)

print("\n--- EXECUTION SAFETY ---")

print(
    "Execution engine unchanged     :",
    engine_sha_now_629aoe
    == FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,
)
print("Model state unchanged          :", True)
print("Model mode restored            :", True)

print(
    "Stuck-block evidence SHA-256   :",
    FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_SHA256_629AOE,
)

print(
    "\nPASS — frozen scenarios 289..336 were executed using the exact "
    "same locked label-free execution engine."
)

print(
    "Full corruption execution progress: 336 / 336 scenarios COMPLETE. "
    "No DEV labels/errors, threshold/policy selection, TEST, or official RUL "
    "was used during execution."
)

print(
    "Next: concatenate and fingerprint all five frozen blocks into one "
    "336-scenario evidence set, verify exact manifest order and block-chain "
    "integrity, and lock the label-free corruption execution before any "
    "DEV outcome/error analysis."
)

In [ ]:
# STEP 6.29AP
# FD003 COMPLETE 336-SCENARIO LABEL-FREE CORRUPTION EVIDENCE LOCK
#
# Concatenates the five already-completed frozen execution blocks and verifies:
#   - exact scenario order 1..336
#   - exact manifest semantic alignment
#   - exact block evidence chain
#   - exact per-row summary <-> retained-array fingerprint linkage
#   - unchanged trained model state
#
# NO new corruption.
# NO model inference.
# NO MC execution.
# NO DEV labels/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aoe", False)):
    raise RuntimeError(
        "STEP 6.29AO-E must pass before STEP 6.29AP."
    )

required_629ap = [
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM",
    "FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN",
    "FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA",

    "FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA",
    "FD003_CORRUPTION_BIAS_MC_STD_629AOA",
    "FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA",
    "FD003_CORRUPTION_BIAS_SUMMARY_629AOA",
    "FD003_CORRUPTION_BIAS_PREDICTIONS_SHA256_629AOA",
    "FD003_CORRUPTION_BIAS_MC_STD_SHA256_629AOA",
    "FD003_CORRUPTION_BIAS_FINAL_RISK_SHA256_629AOA",
    "FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_629AOA",
    "FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA",

    "FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB",
    "FD003_CORRUPTION_NOISE_MC_STD_629AOB",
    "FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB",
    "FD003_CORRUPTION_NOISE_SUMMARY_629AOB",
    "FD003_CORRUPTION_NOISE_PREDICTIONS_SHA256_629AOB",
    "FD003_CORRUPTION_NOISE_MC_STD_SHA256_629AOB",
    "FD003_CORRUPTION_NOISE_FINAL_RISK_SHA256_629AOB",
    "FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_629AOB",
    "FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB",

    "FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC",
    "FD003_CORRUPTION_DRIFT_MC_STD_629AOC",
    "FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC",
    "FD003_CORRUPTION_DRIFT_SUMMARY_629AOC",
    "FD003_CORRUPTION_DRIFT_PREDICTIONS_SHA256_629AOC",
    "FD003_CORRUPTION_DRIFT_MC_STD_SHA256_629AOC",
    "FD003_CORRUPTION_DRIFT_FINAL_RISK_SHA256_629AOC",
    "FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_629AOC",
    "FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC",

    "FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD",
    "FD003_CORRUPTION_DROPOUT_MC_STD_629AOD",
    "FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD",
    "FD003_CORRUPTION_DROPOUT_SUMMARY_629AOD",
    "FD003_CORRUPTION_DROPOUT_PREDICTIONS_SHA256_629AOD",
    "FD003_CORRUPTION_DROPOUT_MC_STD_SHA256_629AOD",
    "FD003_CORRUPTION_DROPOUT_FINAL_RISK_SHA256_629AOD",
    "FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_629AOD",
    "FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD",

    "FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE",
    "FD003_CORRUPTION_STUCK_MC_STD_629AOE",
    "FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE",
    "FD003_CORRUPTION_STUCK_SUMMARY_629AOE",
    "FD003_CORRUPTION_STUCK_PREDICTIONS_SHA256_629AOE",
    "FD003_CORRUPTION_STUCK_MC_STD_SHA256_629AOE",
    "FD003_CORRUPTION_STUCK_FINAL_RISK_SHA256_629AOE",
    "FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_629AOE",
    "FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_SHA256_629AOE",

    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "_state_dict_sha256_629k",
]

missing_629ap = [
    name
    for name in required_629ap
    if name not in globals()
]

if missing_629ap:
    raise RuntimeError(
        "Missing STEP 6.29AP object(s): "
        + ", ".join(missing_629ap)
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AOE", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AP."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AOE", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AP."
    )

if bool(globals().get("FD003_CORRUPTION_FULL_EVIDENCE_LOCKED_629AP", False)):
    raise RuntimeError(
        "STEP 6.29AP has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629ap(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AP hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _json_scalar_629ap(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        f"Unsupported canonical scalar type: {type(value).__name__}"
    )


def _sha_json_629ap(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _dataframe_records_sha256_629ap(df, columns):

    records = []

    for row in df[columns].itertuples(
        index=False,
        name=None,
    ):

        records.append(
            [
                _json_scalar_629ap(x)
                for x in row
            ]
        )

    return hashlib.sha256(
        json.dumps(
            records,
            separators=(",", ":"),
            ensure_ascii=False,
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Block registry
# =============================================================================

blocks_629ap = [
    {
        "name": "bias",
        "start": 1,
        "end": 96,
        "count": 96,
        "pred": FD003_CORRUPTION_BIAS_PREDICTIONS_629AOA,
        "mc": FD003_CORRUPTION_BIAS_MC_STD_629AOA,
        "risk": FD003_CORRUPTION_BIAS_FINAL_RISK_629AOA,
        "summary": FD003_CORRUPTION_BIAS_SUMMARY_629AOA,
        "pred_sha": FD003_CORRUPTION_BIAS_PREDICTIONS_SHA256_629AOA,
        "mc_sha": FD003_CORRUPTION_BIAS_MC_STD_SHA256_629AOA,
        "risk_sha": FD003_CORRUPTION_BIAS_FINAL_RISK_SHA256_629AOA,
        "evidence": FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_629AOA,
        "evidence_sha": FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,
        "expected_previous": None,
    },
    {
        "name": "noise",
        "start": 97,
        "end": 144,
        "count": 48,
        "pred": FD003_CORRUPTION_NOISE_PREDICTIONS_629AOB,
        "mc": FD003_CORRUPTION_NOISE_MC_STD_629AOB,
        "risk": FD003_CORRUPTION_NOISE_FINAL_RISK_629AOB,
        "summary": FD003_CORRUPTION_NOISE_SUMMARY_629AOB,
        "pred_sha": FD003_CORRUPTION_NOISE_PREDICTIONS_SHA256_629AOB,
        "mc_sha": FD003_CORRUPTION_NOISE_MC_STD_SHA256_629AOB,
        "risk_sha": FD003_CORRUPTION_NOISE_FINAL_RISK_SHA256_629AOB,
        "evidence": FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_629AOB,
        "evidence_sha": FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,
        "expected_previous": FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,
    },
    {
        "name": "drift",
        "start": 145,
        "end": 240,
        "count": 96,
        "pred": FD003_CORRUPTION_DRIFT_PREDICTIONS_629AOC,
        "mc": FD003_CORRUPTION_DRIFT_MC_STD_629AOC,
        "risk": FD003_CORRUPTION_DRIFT_FINAL_RISK_629AOC,
        "summary": FD003_CORRUPTION_DRIFT_SUMMARY_629AOC,
        "pred_sha": FD003_CORRUPTION_DRIFT_PREDICTIONS_SHA256_629AOC,
        "mc_sha": FD003_CORRUPTION_DRIFT_MC_STD_SHA256_629AOC,
        "risk_sha": FD003_CORRUPTION_DRIFT_FINAL_RISK_SHA256_629AOC,
        "evidence": FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_629AOC,
        "evidence_sha": FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,
        "expected_previous": FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,
    },
    {
        "name": "dropout",
        "start": 241,
        "end": 288,
        "count": 48,
        "pred": FD003_CORRUPTION_DROPOUT_PREDICTIONS_629AOD,
        "mc": FD003_CORRUPTION_DROPOUT_MC_STD_629AOD,
        "risk": FD003_CORRUPTION_DROPOUT_FINAL_RISK_629AOD,
        "summary": FD003_CORRUPTION_DROPOUT_SUMMARY_629AOD,
        "pred_sha": FD003_CORRUPTION_DROPOUT_PREDICTIONS_SHA256_629AOD,
        "mc_sha": FD003_CORRUPTION_DROPOUT_MC_STD_SHA256_629AOD,
        "risk_sha": FD003_CORRUPTION_DROPOUT_FINAL_RISK_SHA256_629AOD,
        "evidence": FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_629AOD,
        "evidence_sha": FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,
        "expected_previous": FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,
    },
    {
        "name": "stuck",
        "start": 289,
        "end": 336,
        "count": 48,
        "pred": FD003_CORRUPTION_STUCK_PREDICTIONS_629AOE,
        "mc": FD003_CORRUPTION_STUCK_MC_STD_629AOE,
        "risk": FD003_CORRUPTION_STUCK_FINAL_RISK_629AOE,
        "summary": FD003_CORRUPTION_STUCK_SUMMARY_629AOE,
        "pred_sha": FD003_CORRUPTION_STUCK_PREDICTIONS_SHA256_629AOE,
        "mc_sha": FD003_CORRUPTION_STUCK_MC_STD_SHA256_629AOE,
        "risk_sha": FD003_CORRUPTION_STUCK_FINAL_RISK_SHA256_629AOE,
        "evidence": FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_629AOE,
        "evidence_sha": FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_SHA256_629AOE,
        "expected_previous": FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,
    },
]


# =============================================================================
# 4) Verify each block before concatenation
# =============================================================================

block_audit_rows_629ap = []

for block in blocks_629ap:

    name = block["name"]
    count = int(block["count"])

    pred = np.asarray(block["pred"])
    mc = np.asarray(block["mc"])
    risk = np.asarray(block["risk"])
    summary = block["summary"]

    if pred.shape != (count, 4706):
        raise RuntimeError(
            f"{name}: prediction matrix shape changed."
        )

    if mc.shape != (count, 4706):
        raise RuntimeError(
            f"{name}: MC-STD matrix shape changed."
        )

    if risk.shape != (count, 4706):
        raise RuntimeError(
            f"{name}: final-risk matrix shape changed."
        )

    if summary.shape != (count, 10):
        raise RuntimeError(
            f"{name}: summary shape changed."
        )

    pred_hash_exact = bool(
        _sha_arr_629ap(pred)
        ==
        block["pred_sha"]
    )

    mc_hash_exact = bool(
        _sha_arr_629ap(mc)
        ==
        block["mc_sha"]
    )

    risk_hash_exact = bool(
        _sha_arr_629ap(risk)
        ==
        block["risk_sha"]
    )

    if not all(
        [
            pred_hash_exact,
            mc_hash_exact,
            risk_hash_exact,
        ]
    ):
        raise RuntimeError(
            f"{name}: retained matrix fingerprint changed."
        )

    expected_numbers = np.arange(
        block["start"],
        block["end"] + 1,
        dtype=np.int64,
    )

    summary_numbers = summary[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    )

    order_exact = bool(
        np.array_equal(
            summary_numbers,
            expected_numbers,
        )
    )

    if not order_exact:
        raise RuntimeError(
            f"{name}: scenario order changed."
        )

    if not bool(
        (
            summary["family"].astype(str)
            ==
            name
        ).all()
    ):
        raise RuntimeError(
            f"{name}: summary family labels changed."
        )

    evidence = block["evidence"]

    if (
        str(
            evidence[
                "parent_execution_protocol_sha256"
            ]
        )
        !=
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM
    ):
        raise RuntimeError(
            f"{name}: parent execution protocol mismatch."
        )

    if (
        str(
            evidence[
                "execution_engine_sha256"
            ]
        )
        !=
        FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA
    ):
        raise RuntimeError(
            f"{name}: execution-engine fingerprint mismatch."
        )

    if bool(evidence["dev_labels_used"]):
        raise RuntimeError(
            f"{name}: DEV labels unexpectedly marked used."
        )

    if bool(evidence["dev_errors_scored"]):
        raise RuntimeError(
            f"{name}: DEV errors unexpectedly marked scored."
        )

    if bool(evidence["threshold_selected"]):
        raise RuntimeError(
            f"{name}: threshold unexpectedly selected."
        )

    if bool(evidence["policy_selected"]):
        raise RuntimeError(
            f"{name}: policy unexpectedly selected."
        )

    if bool(evidence["test_contents_read"]):
        raise RuntimeError(
            f"{name}: TEST unexpectedly read."
        )

    if bool(evidence["official_rul_contents_read"]):
        raise RuntimeError(
            f"{name}: official RUL unexpectedly read."
        )

    chain_exact = True

    if block["expected_previous"] is not None:

        chain_exact = bool(
            str(
                evidence[
                    "previous_block_evidence_sha256"
                ]
            )
            ==
            str(
                block["expected_previous"]
            )
        )

        if not chain_exact:
            raise RuntimeError(
                f"{name}: previous-block evidence chain mismatch."
            )

    for i in range(count):

        if (
            str(
                summary.iloc[i][
                    "prediction_sha256"
                ]
            )
            !=
            _sha_arr_629ap(
                pred[i]
            )
        ):
            raise RuntimeError(
                f"{name}: prediction row hash mismatch at local row {i}."
            )

        if (
            str(
                summary.iloc[i][
                    "mc_std_sha256"
                ]
            )
            !=
            _sha_arr_629ap(
                mc[i]
            )
        ):
            raise RuntimeError(
                f"{name}: MC-STD row hash mismatch at local row {i}."
            )

        if (
            str(
                summary.iloc[i][
                    "final_risk_sha256"
                ]
            )
            !=
            _sha_arr_629ap(
                risk[i]
            )
        ):
            raise RuntimeError(
                f"{name}: final-risk row hash mismatch at local row {i}."
            )

    block_audit_rows_629ap.append(
        {
            "family": name,
            "scenario_start": int(block["start"]),
            "scenario_end": int(block["end"]),
            "scenario_count": count,
            "prediction_hash_exact": pred_hash_exact,
            "mc_std_hash_exact": mc_hash_exact,
            "final_risk_hash_exact": risk_hash_exact,
            "scenario_order_exact": order_exact,
            "block_chain_exact": chain_exact,
        }
    )


FD003_CORRUPTION_BLOCK_AUDIT_629AP = pd.DataFrame(
    block_audit_rows_629ap
)


# =============================================================================
# 5) Concatenate all 336 scenarios
# =============================================================================

FD003_CORRUPTION_ALL_PREDICTIONS_629AP = np.concatenate(
    [
        np.asarray(block["pred"])
        for block in blocks_629ap
    ],
    axis=0,
).astype(
    np.float64,
    copy=False,
)

FD003_CORRUPTION_ALL_MC_STD_629AP = np.concatenate(
    [
        np.asarray(block["mc"])
        for block in blocks_629ap
    ],
    axis=0,
).astype(
    np.float64,
    copy=False,
)

FD003_CORRUPTION_ALL_FINAL_RISK_629AP = np.concatenate(
    [
        np.asarray(block["risk"])
        for block in blocks_629ap
    ],
    axis=0,
).astype(
    np.float64,
    copy=False,
)

FD003_CORRUPTION_ALL_SUMMARY_629AP = pd.concat(
    [
        block["summary"]
        for block in blocks_629ap
    ],
    axis=0,
    ignore_index=True,
)


if FD003_CORRUPTION_ALL_PREDICTIONS_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Full prediction matrix shape is not (336, 4706)."
    )

if FD003_CORRUPTION_ALL_MC_STD_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Full MC-STD matrix shape is not (336, 4706)."
    )

if FD003_CORRUPTION_ALL_FINAL_RISK_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Full final-risk matrix shape is not (336, 4706)."
    )

if FD003_CORRUPTION_ALL_SUMMARY_629AP.shape != (336, 10):
    raise RuntimeError(
        "Full scenario summary shape is not (336, 10)."
    )

if not np.isfinite(
    FD003_CORRUPTION_ALL_PREDICTIONS_629AP
).all():
    raise RuntimeError(
        "Full prediction evidence contains non-finite values."
    )

if not np.isfinite(
    FD003_CORRUPTION_ALL_MC_STD_629AP
).all():
    raise RuntimeError(
        "Full MC-STD evidence contains non-finite values."
    )

if not np.isfinite(
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP
).all():
    raise RuntimeError(
        "Full final-risk evidence contains non-finite values."
    )

if not bool(
    (
        (FD003_CORRUPTION_ALL_FINAL_RISK_629AP >= 0.0)
        &
        (FD003_CORRUPTION_ALL_FINAL_RISK_629AP <= 1.0)
    ).all()
):
    raise RuntimeError(
        "Full final-risk evidence is outside [0,1]."
    )


# =============================================================================
# 6) Exact manifest semantic alignment
# =============================================================================

manifest_semantic_629ap = (
    FD003_CORRUPTION_MANIFEST_629AF[
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .reset_index(
        drop=True
    )
)

summary_semantic_629ap = (
    FD003_CORRUPTION_ALL_SUMMARY_629AP[
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
        ]
    ]
    .reset_index(
        drop=True
    )
)

manifest_alignment_exact_629ap = bool(
    manifest_semantic_629ap.equals(
        summary_semantic_629ap
    )
)

if not manifest_alignment_exact_629ap:
    raise RuntimeError(
        "Full execution summary does not exactly align with frozen manifest."
    )

if not np.array_equal(
    FD003_CORRUPTION_ALL_SUMMARY_629AP[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    ),
    np.arange(
        1,
        337,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "Full execution scenario order is not exactly 1..336."
    )


# =============================================================================
# 7) Full evidence fingerprints
# =============================================================================

FD003_CORRUPTION_ALL_PREDICTIONS_SHA256_629AP = _sha_arr_629ap(
    FD003_CORRUPTION_ALL_PREDICTIONS_629AP
)

FD003_CORRUPTION_ALL_MC_STD_SHA256_629AP = _sha_arr_629ap(
    FD003_CORRUPTION_ALL_MC_STD_629AP
)

FD003_CORRUPTION_ALL_FINAL_RISK_SHA256_629AP = _sha_arr_629ap(
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP
)

FD003_CORRUPTION_ALL_SUMMARY_SHA256_629AP = (
    _dataframe_records_sha256_629ap(
        FD003_CORRUPTION_ALL_SUMMARY_629AP,
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "prediction_sha256",
            "mc_std_sha256",
            "final_risk_sha256",
            "model_state_unchanged",
            "model_mode_restored",
        ],
    )
)


# =============================================================================
# 8) Current model-state safety check
# =============================================================================

model_sha_current_629ap = _state_dict_sha256_629k(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
)

if (
    model_sha_current_629ap
    !=
    FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
):
    raise RuntimeError(
        "FD003 trained model state changed after the full corruption run."
    )

if bool(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH.training
):
    raise RuntimeError(
        "FD003 trained model is not restored to eval mode."
    )


# =============================================================================
# 9) Full label-free execution lock
# =============================================================================

total_runtime_seconds_629ap = float(
    sum(
        float(
            block["evidence"][
                "runtime_seconds"
            ]
        )
        for block in blocks_629ap
    )
)


FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_629AP = {

    "step":
        "6.29AP",

    "dataset":
        "FD003",

    "parent_execution_protocol_sha256":
        FD003_CORRUPTION_EXECUTION_PROTOCOL_SHA256_629AM,

    "helper_return_contract_sha256":
        FD003_FULL_RUN_HELPER_RETURN_CONTRACT_SHA256_629AN,

    "execution_engine_sha256":
        FD003_CORRUPTION_EXECUTION_ENGINE_SHA256_629AOA,

    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "scenario_count":
        336,

    "scenario_number_range":
        [1, 336],

    "manifest_alignment_exact":
        True,

    "family_blocks":
        [
            {
                "family": str(row["family"]),
                "scenario_start": int(row["scenario_start"]),
                "scenario_end": int(row["scenario_end"]),
                "scenario_count": int(row["scenario_count"]),
                "block_chain_exact": bool(row["block_chain_exact"]),
            }
            for row in block_audit_rows_629ap
        ],

    "block_evidence_sha256":
        {
            "bias":
                FD003_CORRUPTION_BIAS_EXECUTION_EVIDENCE_SHA256_629AOA,

            "noise":
                FD003_CORRUPTION_NOISE_EXECUTION_EVIDENCE_SHA256_629AOB,

            "drift":
                FD003_CORRUPTION_DRIFT_EXECUTION_EVIDENCE_SHA256_629AOC,

            "dropout":
                FD003_CORRUPTION_DROPOUT_EXECUTION_EVIDENCE_SHA256_629AOD,

            "stuck":
                FD003_CORRUPTION_STUCK_EXECUTION_EVIDENCE_SHA256_629AOE,
        },

    "prediction_matrix_shape":
        [336, 4706],

    "prediction_matrix_sha256":
        FD003_CORRUPTION_ALL_PREDICTIONS_SHA256_629AP,

    "mc_std_matrix_shape":
        [336, 4706],

    "mc_std_matrix_sha256":
        FD003_CORRUPTION_ALL_MC_STD_SHA256_629AP,

    "final_risk_matrix_shape":
        [336, 4706],

    "final_risk_matrix_sha256":
        FD003_CORRUPTION_ALL_FINAL_RISK_SHA256_629AP,

    "summary_sha256":
        FD003_CORRUPTION_ALL_SUMMARY_SHA256_629AP,

    "total_block_runtime_seconds":
        total_runtime_seconds_629ap,

    "model_state_sha256":
        model_sha_current_629ap,

    "model_state_unchanged":
        True,

    "model_mode_restored_eval":
        True,

    "dev_labels_used":
        False,

    "dev_errors_scored":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "next_stage":
        "freeze a separate DEV-only corruption outcome/risk scoring protocol "
        "before inspecting corrupted prediction errors",
}


FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP = (
    _sha_json_629ap(
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_629AP
    )
)


for arr in (
    FD003_CORRUPTION_ALL_PREDICTIONS_629AP,
    FD003_CORRUPTION_ALL_MC_STD_629AP,
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP,
):
    arr.setflags(
        write=False
    )


FD003_CORRUPTION_FULL_EVIDENCE_LOCKED_629AP = True
FD003_CORRUPTION_FULL_RUN_COMPLETE_629AP = True
FD003_CORRUPTION_SCENARIOS_EXECUTED_629AP = 336

FD003_DEV_LABELS_USED_629AP = False
FD003_DEV_SCORING_PERFORMED_629AP = False
FD003_THRESHOLD_SELECTED_629AP = False
FD003_POLICY_SELECTED_629AP = False
FD003_TEST_CONTENTS_READ_629AP = False
FD003_RUL_CONTENTS_READ_629AP = False

all_checks_passed_629ap = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 156)

print(
    "STEP 6.29AP — FD003 COMPLETE 336-SCENARIO "
    "LABEL-FREE CORRUPTION EVIDENCE LOCK"
)

print("=" * 156)

print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("DEV labels used                :", False)
print("DEV error scoring performed    :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FIVE-BLOCK INTEGRITY AUDIT ---"
)

print(
    FD003_CORRUPTION_BLOCK_AUDIT_629AP.to_string(
        index=False
    )
)

print(
    "\n--- COMPLETE LABEL-FREE EVIDENCE ---"
)

print(
    "Scenario count                 :",
    336,
)

print(
    "Manifest alignment exact       :",
    manifest_alignment_exact_629ap,
)

print(
    "Prediction matrix shape        :",
    FD003_CORRUPTION_ALL_PREDICTIONS_629AP.shape,
)

print(
    "MC-STD matrix shape            :",
    FD003_CORRUPTION_ALL_MC_STD_629AP.shape,
)

print(
    "Final-risk matrix shape        :",
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP.shape,
)

print(
    "Prediction matrix SHA-256      :",
    FD003_CORRUPTION_ALL_PREDICTIONS_SHA256_629AP,
)

print(
    "MC-STD matrix SHA-256          :",
    FD003_CORRUPTION_ALL_MC_STD_SHA256_629AP,
)

print(
    "Final-risk matrix SHA-256      :",
    FD003_CORRUPTION_ALL_FINAL_RISK_SHA256_629AP,
)

print(
    "Full summary SHA-256           :",
    FD003_CORRUPTION_ALL_SUMMARY_SHA256_629AP,
)

print(
    "\n--- FINAL EXECUTION SAFETY ---"
)

print(
    "Model state unchanged          :",
    True,
)

print(
    "Model restored to eval         :",
    True,
)

print(
    "Complete execution locked      :",
    True,
)

print(
    "Full-execution contract SHA-256:",
    FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP,
)

print(
    "\nPASS — all 336 frozen FD003 corruption scenarios are now "
    "concatenated, manifest-aligned, fingerprinted, chain-verified, "
    "and locked as one complete label-free corruption evidence set."
)

print(
    "No DEV outcome/error analysis has yet been performed. "
    "Next: prospectively freeze the DEV-only corruption scoring protocol "
    "before using DEV RUL outcomes to evaluate robustness and reliability."
)

In [ ]:
# STEP 6.29AQ
# FD003 DEV-ONLY CORRUPTION OUTCOME / RELIABILITY SCORING-PROTOCOL FREEZE
#
# Prospective scoring freeze BEFORE inspecting any corrupted prediction error.
#
# NO DEV target values are read in this step.
# NO corrupted errors are computed.
# NO new corruption / inference / MC execution.
# NO threshold / operating policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ap", False)):
    raise RuntimeError(
        "STEP 6.29AP must pass before STEP 6.29AQ."
    )

required_629aq = [
    "FD003_CORRUPTION_FULL_EVIDENCE_LOCKED_629AP",
    "FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_629AP",
    "FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP",
    "FD003_CORRUPTION_ALL_PREDICTIONS_629AP",
    "FD003_CORRUPTION_ALL_PREDICTIONS_SHA256_629AP",
    "FD003_CORRUPTION_ALL_MC_STD_629AP",
    "FD003_CORRUPTION_ALL_MC_STD_SHA256_629AP",
    "FD003_CORRUPTION_ALL_FINAL_RISK_629AP",
    "FD003_CORRUPTION_ALL_FINAL_RISK_SHA256_629AP",
    "FD003_CORRUPTION_ALL_SUMMARY_629AP",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_MANIFEST_SHA256_629AF",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
]

missing_629aq = [
    name
    for name in required_629aq
    if name not in globals()
]

if missing_629aq:
    raise RuntimeError(
        "Missing STEP 6.29AQ object(s): "
        + ", ".join(missing_629aq)
    )

if not bool(FD003_CORRUPTION_FULL_EVIDENCE_LOCKED_629AP):
    raise RuntimeError(
        "The complete label-free corruption evidence is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AP", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AQ."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AP", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AQ."
    )

if bool(
    globals().get(
        "FD003_CORRUPTION_SCORING_PROTOCOL_LOCKED_629AQ",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AQ has already been completed."
    )


# =============================================================================
# 2) Hash helpers — evidence identity only, no outcome access
# =============================================================================

def _sha_arr_629aq(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AQ hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629aq(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify frozen label-free evidence
# =============================================================================

if (
    _sha_arr_629aq(
        FD003_CORRUPTION_ALL_PREDICTIONS_629AP
    )
    !=
    FD003_CORRUPTION_ALL_PREDICTIONS_SHA256_629AP
):
    raise RuntimeError(
        "Full corruption prediction matrix changed before scoring freeze."
    )

if (
    _sha_arr_629aq(
        FD003_CORRUPTION_ALL_MC_STD_629AP
    )
    !=
    FD003_CORRUPTION_ALL_MC_STD_SHA256_629AP
):
    raise RuntimeError(
        "Full corruption MC-STD matrix changed before scoring freeze."
    )

if (
    _sha_arr_629aq(
        FD003_CORRUPTION_ALL_FINAL_RISK_629AP
    )
    !=
    FD003_CORRUPTION_ALL_FINAL_RISK_SHA256_629AP
):
    raise RuntimeError(
        "Full corruption final-risk matrix changed before scoring freeze."
    )

if (
    _sha_json_629aq(
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_629AP
    )
    !=
    FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP
):
    raise RuntimeError(
        "Full corruption execution contract changed before scoring freeze."
    )

if FD003_CORRUPTION_ALL_PREDICTIONS_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Frozen prediction matrix shape changed."
    )

if FD003_CORRUPTION_ALL_FINAL_RISK_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Frozen risk matrix shape changed."
    )


# =============================================================================
# 4) Freeze exact evaluation population and ordering rules
# =============================================================================

n_scenarios_629aq = 336
n_windows_629aq = 4706

diagnostic_coverages_629aq = [
    0.9,
    0.8,
    0.7,
]

accepted_counts_629aq = {
    str(cov):
        int(
            np.ceil(
                cov
                *
                n_windows_629aq
            )
        )
    for cov in diagnostic_coverages_629aq
}

expected_counts_629aq = {
    "0.9": 4236,
    "0.8": 3765,
    "0.7": 3295,
}

if accepted_counts_629aq != expected_counts_629aq:
    raise RuntimeError(
        "Frozen diagnostic coverage counts are inconsistent."
    )


# =============================================================================
# 5) Prospectively frozen corrupted-DEV scoring protocol
# =============================================================================

FD003_CORRUPTION_SCORING_PROTOCOL_629AQ = {

    "step":
        "6.29AQ",

    "dataset":
        "FD003",

    "purpose":
        "DEV-only controlled-corruption robustness and reliability evaluation",

    "evidence_source":
        "STEP 6.29AP complete 336-scenario label-free corruption evidence",

    "full_execution_contract_sha256":
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP,

    "manifest_sha256":
        FD003_CORRUPTION_MANIFEST_SHA256_629AF,

    "scenario_count":
        n_scenarios_629aq,

    "windows_per_scenario":
        n_windows_629aq,

    "scenario_order":
        "ascending frozen scenario_number 1..336",

    "outcome_source":
        "frozen FD003 DEV window RUL targets from STEP 6.29F; "
        "one identical target vector is used for every corruption scenario",

    "outcome_alignment_rule":
        "same 4706 frozen DEV window unit/cycle keys already verified "
        "unchanged in the corruption execution bridge",

    "corrupted_absolute_error":
        "abs(corrupted_deterministic_prediction - frozen_DEV_RUL_target)",

    "corrupted_squared_error":
        "(corrupted_deterministic_prediction - frozen_DEV_RUL_target)^2",

    "per_scenario_full_error_metrics":
        [
            "MAE",
            "median_absolute_error",
            "RMSE",
        ],

    "primary_reliability_ordering_metric":
        "Spearman(final_selective_risk, absolute_RUL_error)",

    "spearman_tie_handling":
        "standard average ranks; report NaN only if either vector is constant",

    "risk_orientation":
        "lower final selective risk = more reliable = accepted first",

    "diagnostic_coverages":
        list(
            diagnostic_coverages_629aq
        ),

    "accepted_counts":
        dict(
            accepted_counts_629aq
        ),

    "coverage_count_rule":
        "ceil(target_coverage * 4706)",

    "selective_order_rule":
        "stable ascending final_selective_risk using mergesort; "
        "original frozen DEV order breaks exact-risk ties",

    "per_coverage_metrics":
        [
            "realized_coverage",
            "accepted_MAE",
            "abstained_MAE",
            "selective_MAE_gain = full_MAE - accepted_MAE",
            "relative_MAE_reduction_pct = 100 * selective_MAE_gain / full_MAE",
            "abstained_minus_accepted_MAE",
        ],

    "primary_selective_operating_diagnostic":
        "target coverage 0.8; diagnostic only, not an operating threshold",

    "clean_comparator":
        "already-frozen clean DEV deterministic prediction and final selective risk; "
        "clean outcomes were evaluated before corruption execution",

    "per_scenario_clean_delta_metrics":
        [
            "delta_full_MAE_vs_clean",
            "ratio_full_MAE_to_clean",
            "delta_Spearman_vs_clean",
            "delta_accepted_MAE_vs_clean_at_0.8",
            "delta_selective_MAE_gain_vs_clean_at_0.8",
        ],

    "label_free_response_descriptors":
        [
            "mean_absolute_prediction_shift_vs_clean",
            "mean_MC_std",
            "mean_final_selective_risk",
            "mean_final_selective_risk_shift_vs_clean",
        ],

    "scenario_level_table_key":
        "scenario_number",

    "predeclared_group_summaries":
        [
            "family",
            "family + severity",
            "sensor",
            "family + sensor",
        ],

    "group_aggregation_rule":
        "macro over scenarios: arithmetic mean, median, minimum, maximum; "
        "do not pool all windows across scenarios as the primary group summary",

    "severity_analysis":
        "within each family/sensor structure, report low/medium/high scenario metrics; "
        "for bias and drift retain direction as a separate manifest attribute "
        "rather than averaging directions before scenario scoring",

    "primary_corruption_robustness_evidence":
        [
            "scenario-wise full MAE degradation vs clean",
            "scenario-wise Spearman(final risk, absolute error)",
            "scenario-wise selective MAE gain at target coverage 0.8",
        ],

    "secondary_corruption_evidence":
        [
            "diagnostic coverage 0.9",
            "diagnostic coverage 0.7",
            "family/severity/sensor macro summaries",
            "label-free prediction/uncertainty/risk shifts",
        ],

    "no_binary_pass_fail_rule":
        True,

    "interpretation_rule":
        "corruption scoring is robustness/reporting evidence; "
        "it must not retrain the model, alter uncertainty/risk construction, "
        "or retrospectively change the 336-scenario manifest",

    "operating_threshold_selection_rule":
        "NO operating threshold is selected in corruption scoring; "
        "threshold selection requires a separate prospectively frozen step",

    "corruption_results_may_modify_model_or_risk_architecture":
        False,

    "corruption_results_may_modify_manifest":
        False,

    "model_retraining_allowed":
        False,

    "new_model_inference_allowed":
        False,

    "new_mc_execution_allowed":
        False,

    "dev_targets_read_in_this_freeze_step":
        False,

    "dev_errors_computed_in_this_freeze_step":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ = _sha_json_629aq(
    FD003_CORRUPTION_SCORING_PROTOCOL_629AQ
)


# =============================================================================
# 6) Lock state
# =============================================================================

FD003_CORRUPTION_SCORING_PROTOCOL_LOCKED_629AQ = True

FD003_DEV_TARGETS_READ_629AQ = False
FD003_CORRUPTED_ERRORS_COMPUTED_629AQ = False
FD003_NEW_CORRUPTION_EXECUTED_629AQ = False
FD003_MODEL_INFERENCE_PERFORMED_629AQ = False
FD003_MC_EXECUTION_PERFORMED_629AQ = False
FD003_THRESHOLD_SELECTED_629AQ = False
FD003_POLICY_SELECTED_629AQ = False
FD003_TEST_CONTENTS_READ_629AQ = False
FD003_RUL_CONTENTS_READ_629AQ = False

all_checks_passed_629aq = True


# =============================================================================
# 7) Output
# =============================================================================

print("=" * 158)

print(
    "STEP 6.29AQ — FD003 DEV-ONLY CORRUPTION "
    "OUTCOME / RELIABILITY SCORING-PROTOCOL FREEZE"
)

print("=" * 158)

print("DEV target values read         :", False)
print("Corrupted errors computed      :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FROZEN CORRUPTION SCORING DESIGN ---"
)

print(
    "Scenarios                      :",
    n_scenarios_629aq,
)

print(
    "DEV windows / scenario         :",
    n_windows_629aq,
)

print(
    "Primary ordering metric        : "
    "Spearman(final risk, absolute RUL error)"
)

print(
    "Diagnostic coverages           :",
    diagnostic_coverages_629aq,
)

print(
    "Accepted counts                :",
    accepted_counts_629aq,
)

print(
    "Primary selective diagnostic   :",
    "coverage 0.8",
)

print(
    "Selection/tie rule             : "
    "stable ascending risk / original DEV order"
)

print(
    "Clean-comparison metrics       : "
    "error degradation + reliability/selectivity deltas"
)

print(
    "Group summaries                : "
    "family, family+severity, sensor, family+sensor"
)

print(
    "Group aggregation              : "
    "scenario-level macro summaries"
)

print(
    "\nScoring-protocol SHA-256       :",
    FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ,
)

print(
    "\nPASS — the corrupted-DEV outcome/reliability scoring protocol "
    "is now prospectively frozen before reading corrupted prediction errors."
)

print(
    "Next: identify and verify the exact frozen DEV target vector, then "
    "execute this protocol once against the already-locked 336-scenario "
    "prediction/risk evidence. No threshold or policy selection will occur."
)

In [ ]:
# STEP 6.29AR
# FD003 FROZEN DEV-TARGET AUTHORITY + ALIGNMENT LOCK
#
# First controlled DEV-outcome access after STEP 6.29AQ scoring freeze.
#
# Reads and fingerprints ONLY the already-frozen DEV window target vector.
# Re-confirms its alignment against the already-established clean DEV baseline.
#
# NO corrupted-scenario errors are computed.
# NO new corruption / inference / MC execution.
# NO threshold / policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aq", False)):
    raise RuntimeError(
        "STEP 6.29AQ must pass before STEP 6.29AR."
    )

required_629ar = [
    "FD003_CORRUPTION_SCORING_PROTOCOL_629AQ",
    "FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ",
    "y_DEV_FD003_629F",
    "DEV_WINDOW_UNIT_IDS_FD003_629F",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L",
    "FD003_CORRUPTION_ALL_PREDICTIONS_629AP",
    "FD003_CORRUPTION_ALL_FINAL_RISK_629AP",
    "FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP",
]

missing_629ar = [
    name
    for name in required_629ar
    if name not in globals()
]

if missing_629ar:
    raise RuntimeError(
        "Missing STEP 6.29AR object(s): "
        + ", ".join(missing_629ar)
    )

if not bool(
    globals().get(
        "FD003_CORRUPTION_SCORING_PROTOCOL_LOCKED_629AQ",
        False,
    )
):
    raise RuntimeError(
        "Corruption scoring protocol is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AQ", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AR."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AQ", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AR."
    )

if bool(globals().get("FD003_DEV_TARGET_AUTHORITY_LOCKED_629AR", False)):
    raise RuntimeError(
        "STEP 6.29AR has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629ar(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AR hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629ar(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify the prospectively frozen scoring protocol
# =============================================================================

if (
    _sha_json_629ar(
        FD003_CORRUPTION_SCORING_PROTOCOL_629AQ
    )
    !=
    FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ
):
    raise RuntimeError(
        "Frozen corruption scoring protocol changed before target access."
    )

if (
    FD003_CORRUPTION_SCORING_PROTOCOL_629AQ[
        "outcome_source"
    ]
    !=
    "frozen FD003 DEV window RUL targets from STEP 6.29F; "
    "one identical target vector is used for every corruption scenario"
):
    raise RuntimeError(
        "Frozen corruption outcome-source contract changed."
    )


# =============================================================================
# 4) Lock exact DEV target vector
# =============================================================================

target_raw_629ar = np.asarray(
    y_DEV_FD003_629F
)

if target_raw_629ar.shape != (4706,):
    raise RuntimeError(
        "Frozen DEV target vector shape is not (4706,)."
    )

if str(target_raw_629ar.dtype) != "float32":
    raise RuntimeError(
        "Frozen DEV target vector dtype changed from float32."
    )

if not np.isfinite(target_raw_629ar).all():
    raise RuntimeError(
        "Frozen DEV target vector contains non-finite values."
    )

if float(np.min(target_raw_629ar)) != 0.0:
    raise RuntimeError(
        "Frozen DEV target minimum changed."
    )

if float(np.max(target_raw_629ar)) != 495.0:
    raise RuntimeError(
        "Frozen DEV target maximum changed."
    )

zero_target_count_629ar = int(
    np.count_nonzero(
        target_raw_629ar == 0.0
    )
)

if zero_target_count_629ar != 20:
    raise RuntimeError(
        "Frozen DEV zero-RUL endpoint count changed."
    )

FD003_DEV_TARGET_VECTOR_629AR = np.array(
    target_raw_629ar,
    dtype=np.float32,
    copy=True,
)

FD003_DEV_TARGET_VECTOR_SHA256_629AR = _sha_arr_629ar(
    FD003_DEV_TARGET_VECTOR_629AR
)


# =============================================================================
# 5) Lock target alignment keys
# =============================================================================

unit_ids_629ar = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD003_629F,
    dtype=np.int64,
).reshape(-1)

endpoint_cycles_629ar = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD003_629F,
    dtype=np.int64,
).reshape(-1)

if unit_ids_629ar.shape != (4706,):
    raise RuntimeError(
        "Frozen DEV window-unit vector shape changed."
    )

if endpoint_cycles_629ar.shape != (4706,):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle vector shape changed."
    )

if len(np.unique(unit_ids_629ar)) != 20:
    raise RuntimeError(
        "Frozen DEV window membership no longer spans 20 engines."
    )

FD003_DEV_TARGET_UNIT_IDS_SHA256_629AR = _sha_arr_629ar(
    unit_ids_629ar
)

FD003_DEV_TARGET_ENDPOINT_CYCLES_SHA256_629AR = _sha_arr_629ar(
    endpoint_cycles_629ar
)


# =============================================================================
# 6) Re-confirm already-established clean baseline only
# =============================================================================

clean_pred_629ar = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)

if clean_pred_629ar.shape != (4706,):
    raise RuntimeError(
        "Frozen clean DEV prediction shape changed."
    )

expected_clean_prediction_sha_629ar = str(
    FD003_DETERMINISTIC_DEV_BASELINE_CONTRACT_629L[
        "prediction_sha256"
    ]
)

if (
    _sha_arr_629ar(
        clean_pred_629ar
    )
    !=
    expected_clean_prediction_sha_629ar
):
    raise RuntimeError(
        "Frozen clean DEV prediction fingerprint changed."
    )

clean_abs_error_629ar = np.abs(
    clean_pred_629ar
    -
    FD003_DEV_TARGET_VECTOR_629AR.astype(
        np.float64
    )
)

clean_mae_629ar = float(
    np.mean(
        clean_abs_error_629ar
    )
)

expected_clean_mae_629ar = 38.07509091407808

if not np.isclose(
    clean_mae_629ar,
    expected_clean_mae_629ar,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Frozen DEV target vector does not reproduce the established "
        "clean DEV MAE."
    )

FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_629AR = np.array(
    clean_abs_error_629ar,
    dtype=np.float64,
    copy=True,
)

FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_SHA256_629AR = _sha_arr_629ar(
    FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_629AR
)

optional_clean_error_match_629ar = None
optional_clean_error_source_629ar = None

for candidate_name_629ar in (
    "FD003_CLEAN_DEV_ABS_ERROR_629L",
    "FD003_CLEAN_DEV_ABS_ERROR_629X",
):

    if candidate_name_629ar in globals():

        candidate_629ar = np.asarray(
            globals()[candidate_name_629ar],
            dtype=np.float64,
        ).reshape(-1)

        if candidate_629ar.shape != (4706,):
            raise RuntimeError(
                candidate_name_629ar
                + " has an unexpected shape."
            )

        optional_clean_error_match_629ar = bool(
            np.array_equal(
                candidate_629ar,
                FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_629AR,
            )
        )

        optional_clean_error_source_629ar = candidate_name_629ar

        if not optional_clean_error_match_629ar:
            raise RuntimeError(
                "Reconfirmed clean absolute errors differ from "
                + candidate_name_629ar
            )

        break


# =============================================================================
# 7) Confirm corrupted evidence remains untouched
# =============================================================================

if FD003_CORRUPTION_ALL_PREDICTIONS_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Locked corruption prediction evidence shape changed."
    )

if FD003_CORRUPTION_ALL_FINAL_RISK_629AP.shape != (336, 4706):
    raise RuntimeError(
        "Locked corruption risk evidence shape changed."
    )


# =============================================================================
# 8) Freeze DEV-target authority contract
# =============================================================================

FD003_DEV_TARGET_AUTHORITY_CONTRACT_629AR = {

    "step":
        "6.29AR",

    "dataset":
        "FD003",

    "scoring_protocol_sha256":
        FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ,

    "full_corruption_execution_contract_sha256":
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP,

    "authoritative_target_object":
        "y_DEV_FD003_629F",

    "target_shape":
        [4706],

    "target_dtype":
        "float32",

    "target_sha256":
        FD003_DEV_TARGET_VECTOR_SHA256_629AR,

    "target_min":
        0.0,

    "target_max":
        495.0,

    "zero_rul_endpoint_count":
        zero_target_count_629ar,

    "window_unit_ids_sha256":
        FD003_DEV_TARGET_UNIT_IDS_SHA256_629AR,

    "window_endpoint_cycles_sha256":
        FD003_DEV_TARGET_ENDPOINT_CYCLES_SHA256_629AR,

    "clean_baseline_mae_reconfirmed":
        clean_mae_629ar,

    "clean_abs_error_sha256":
        FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_SHA256_629AR,

    "optional_existing_clean_error_source":
        optional_clean_error_source_629ar,

    "optional_existing_clean_error_exact":
        optional_clean_error_match_629ar,

    "dev_target_values_read":
        True,

    "corrupted_errors_computed":
        False,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR = _sha_json_629ar(
    FD003_DEV_TARGET_AUTHORITY_CONTRACT_629AR
)


# =============================================================================
# 9) Read-only lock
# =============================================================================

FD003_DEV_TARGET_VECTOR_629AR.setflags(
    write=False
)

FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_629AR.setflags(
    write=False
)

FD003_DEV_TARGET_AUTHORITY_LOCKED_629AR = True
FD003_DEV_TARGETS_READ_629AR = True
FD003_CORRUPTED_ERRORS_COMPUTED_629AR = False
FD003_NEW_CORRUPTION_EXECUTED_629AR = False
FD003_MODEL_INFERENCE_PERFORMED_629AR = False
FD003_MC_EXECUTION_PERFORMED_629AR = False
FD003_THRESHOLD_SELECTED_629AR = False
FD003_POLICY_SELECTED_629AR = False
FD003_TEST_CONTENTS_READ_629AR = False
FD003_RUL_CONTENTS_READ_629AR = False

all_checks_passed_629ar = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 158)

print(
    "STEP 6.29AR — FD003 FROZEN DEV-TARGET "
    "AUTHORITY + ALIGNMENT LOCK"
)

print("=" * 158)

print("DEV target values read         :", True)
print("Corrupted errors computed      :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- DEV TARGET AUTHORITY ---")

print(
    "Authoritative target object    :",
    "y_DEV_FD003_629F",
)

print(
    "Target shape / dtype           :",
    (
        FD003_DEV_TARGET_VECTOR_629AR.shape,
        FD003_DEV_TARGET_VECTOR_629AR.dtype,
    ),
)

print(
    "Target min / max               :",
    (
        float(np.min(FD003_DEV_TARGET_VECTOR_629AR)),
        float(np.max(FD003_DEV_TARGET_VECTOR_629AR)),
    ),
)

print(
    "Zero-RUL endpoints             :",
    zero_target_count_629ar,
)

print(
    "Target SHA-256                 :",
    FD003_DEV_TARGET_VECTOR_SHA256_629AR,
)

print(
    "Window-unit SHA-256            :",
    FD003_DEV_TARGET_UNIT_IDS_SHA256_629AR,
)

print(
    "Endpoint-cycle SHA-256         :",
    FD003_DEV_TARGET_ENDPOINT_CYCLES_SHA256_629AR,
)

print("\n--- CLEAN BASELINE ALIGNMENT RECONFIRMATION ---")

print(
    "Reconfirmed clean DEV MAE      :",
    clean_mae_629ar,
)

print(
    "Expected frozen clean DEV MAE  :",
    expected_clean_mae_629ar,
)

print(
    "Clean MAE alignment exact      :",
    bool(
        np.isclose(
            clean_mae_629ar,
            expected_clean_mae_629ar,
            rtol=0.0,
            atol=1e-12,
        )
    ),
)

print(
    "Clean abs-error SHA-256        :",
    FD003_CLEAN_DEV_ABS_ERROR_RECONFIRMED_SHA256_629AR,
)

print(
    "Existing clean-error source    :",
    optional_clean_error_source_629ar,
)

print(
    "Existing clean-error exact     :",
    optional_clean_error_match_629ar,
)

print(
    "\nDEV-target authority contract SHA-256:",
    FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR,
)

print(
    "\nPASS — the exact frozen FD003 DEV target vector is now "
    "fingerprinted and alignment-verified against the established clean "
    "DEV baseline."
)

print(
    "No corrupted prediction error has yet been computed. "
    "Next: execute the already-frozen STEP 6.29AQ scoring protocol once "
    "against the locked 336-scenario prediction/risk evidence."
)

In [ ]:
# STEP 6.29AS
# FD003 FROZEN 336-SCENARIO DEV CORRUPTION OUTCOME / RELIABILITY SCORING
#
# Executes the prospectively frozen STEP 6.29AQ scoring protocol ONCE
# against the already-locked STEP 6.29AP label-free evidence and
# STEP 6.29AR authoritative DEV target vector.
#
# NO new corruption / inference / MC execution.
# NO model retraining.
# NO threshold / operating-policy selection.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ar", False)):
    raise RuntimeError(
        "STEP 6.29AR must pass before STEP 6.29AS."
    )

required_629as = [
    "FD003_CORRUPTION_SCORING_PROTOCOL_629AQ",
    "FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ",
    "FD003_DEV_TARGET_VECTOR_629AR",
    "FD003_DEV_TARGET_VECTOR_SHA256_629AR",
    "FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CORRUPTION_ALL_PREDICTIONS_629AP",
    "FD003_CORRUPTION_ALL_MC_STD_629AP",
    "FD003_CORRUPTION_ALL_FINAL_RISK_629AP",
    "FD003_CORRUPTION_ALL_SUMMARY_629AP",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP",
]

missing_629as = [
    name
    for name in required_629as
    if name not in globals()
]

if missing_629as:
    raise RuntimeError(
        "Missing STEP 6.29AS object(s): "
        + ", ".join(missing_629as)
    )

if not bool(
    globals().get(
        "FD003_CORRUPTION_SCORING_PROTOCOL_LOCKED_629AQ",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AQ scoring protocol is not locked."
    )

if not bool(
    globals().get(
        "FD003_DEV_TARGET_AUTHORITY_LOCKED_629AR",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AR DEV-target authority is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AR", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AS."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AR", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AS."
    )

if bool(
    globals().get(
        "FD003_CORRUPTION_SCORING_COMPLETE_629AS",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AS has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629as(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AS hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _json_scalar_629as(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        "Unsupported canonical scalar type: "
        + type(value).__name__
    )


def _sha_json_629as(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _df_sha256_629as(df):

    columns = [
        str(c)
        for c in df.columns
    ]

    records = []

    for row in df.itertuples(
        index=False,
        name=None,
    ):
        records.append(
            [
                _json_scalar_629as(x)
                for x in row
            ]
        )

    return _sha_json_629as(
        {
            "columns": columns,
            "records": records,
        }
    )


def _average_ranks_629as(values):

    x = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    n = x.size

    order = np.argsort(
        x,
        kind="mergesort",
    )

    sorted_x = x[
        order
    ]

    ranks = np.empty(
        n,
        dtype=np.float64,
    )

    i = 0

    while i < n:

        j = i + 1

        while (
            j < n
            and
            sorted_x[j] == sorted_x[i]
        ):
            j += 1

        average_rank = (
            (i + 1)
            +
            j
        ) / 2.0

        ranks[
            order[i:j]
        ] = average_rank

        i = j

    return ranks


def _spearman_629as(x, y):

    rx = _average_ranks_629as(x)
    ry = _average_ranks_629as(y)

    rx_centered = rx - np.mean(rx)
    ry_centered = ry - np.mean(ry)

    denom = float(
        np.sqrt(
            np.sum(rx_centered ** 2)
            *
            np.sum(ry_centered ** 2)
        )
    )

    if denom == 0.0:
        return float("nan")

    return float(
        np.sum(
            rx_centered
            *
            ry_centered
        )
        /
        denom
    )


def _selective_metrics_629as(
    abs_error,
    risk,
    accepted_count,
):

    err = np.asarray(
        abs_error,
        dtype=np.float64,
    ).reshape(-1)

    r = np.asarray(
        risk,
        dtype=np.float64,
    ).reshape(-1)

    if err.shape != (4706,):
        raise RuntimeError(
            "Selective error vector shape changed."
        )

    if r.shape != (4706,):
        raise RuntimeError(
            "Selective risk vector shape changed."
        )

    k = int(
        accepted_count
    )

    order = np.argsort(
        r,
        kind="mergesort",
    )

    accepted_idx = order[
        :k
    ]

    abstained_idx = order[
        k:
    ]

    full_mae = float(
        np.mean(err)
    )

    accepted_mae = float(
        np.mean(
            err[
                accepted_idx
            ]
        )
    )

    abstained_mae = float(
        np.mean(
            err[
                abstained_idx
            ]
        )
    )

    gain = float(
        full_mae
        -
        accepted_mae
    )

    return {
        "accepted_count":
            k,

        "realized_coverage":
            float(
                k
                /
                len(err)
            ),

        "accepted_mae":
            accepted_mae,

        "abstained_mae":
            abstained_mae,

        "selective_mae_gain":
            gain,

        "relative_mae_reduction_pct":
            float(
                100.0
                *
                gain
                /
                full_mae
            ),

        "abstained_minus_accepted_mae":
            float(
                abstained_mae
                -
                accepted_mae
            ),
    }


# =============================================================================
# 3) Re-certify scoring/target/evidence identities
# =============================================================================

if (
    _sha_json_629as(
        FD003_CORRUPTION_SCORING_PROTOCOL_629AQ
    )
    !=
    FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ
):
    raise RuntimeError(
        "Frozen STEP 6.29AQ scoring protocol changed."
    )

target_629as = np.asarray(
    FD003_DEV_TARGET_VECTOR_629AR,
    dtype=np.float64,
).reshape(-1)

if target_629as.shape != (4706,):
    raise RuntimeError(
        "Authoritative DEV target shape changed."
    )

if (
    _sha_arr_629as(
        np.asarray(
            FD003_DEV_TARGET_VECTOR_629AR
        )
    )
    !=
    FD003_DEV_TARGET_VECTOR_SHA256_629AR
):
    raise RuntimeError(
        "Authoritative DEV target fingerprint changed."
    )

pred_all_629as = np.asarray(
    FD003_CORRUPTION_ALL_PREDICTIONS_629AP,
    dtype=np.float64,
)

mc_all_629as = np.asarray(
    FD003_CORRUPTION_ALL_MC_STD_629AP,
    dtype=np.float64,
)

risk_all_629as = np.asarray(
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP,
    dtype=np.float64,
)

if pred_all_629as.shape != (336, 4706):
    raise RuntimeError(
        "Locked corruption prediction matrix shape changed."
    )

if mc_all_629as.shape != (336, 4706):
    raise RuntimeError(
        "Locked corruption MC-STD matrix shape changed."
    )

if risk_all_629as.shape != (336, 4706):
    raise RuntimeError(
        "Locked corruption risk matrix shape changed."
    )


# =============================================================================
# 4) Clean comparator under the exact frozen scoring rules
# =============================================================================

clean_pred_629as = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)

clean_risk_629as = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
    dtype=np.float64,
).reshape(-1)

clean_abs_629as = np.abs(
    clean_pred_629as
    -
    target_629as
)

clean_full_mae_629as = float(
    np.mean(clean_abs_629as)
)

clean_median_abs_629as = float(
    np.median(clean_abs_629as)
)

clean_rmse_629as = float(
    np.sqrt(
        np.mean(
            (
                clean_pred_629as
                -
                target_629as
            ) ** 2
        )
    )
)

clean_spearman_629as = _spearman_629as(
    clean_risk_629as,
    clean_abs_629as,
)

if not np.isclose(
    clean_full_mae_629as,
    38.07509091407808,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Clean full MAE no longer reproduces the established baseline."
    )

if not np.isclose(
    clean_spearman_629as,
    0.1143894776748529,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "Clean Spearman no longer reproduces STEP 6.29X."
    )

coverage_spec_629as = [
    (0.9, 4236, "0p9"),
    (0.8, 3765, "0p8"),
    (0.7, 3295, "0p7"),
]

clean_selective_629as = {}

for coverage, count, tag in coverage_spec_629as:

    clean_selective_629as[
        tag
    ] = _selective_metrics_629as(
        clean_abs_629as,
        clean_risk_629as,
        count,
    )

FD003_CORRUPTION_SCORING_CLEAN_REFERENCE_629AS = pd.DataFrame(
    [
        {
            "target_coverage": coverage,
            **clean_selective_629as[tag],
        }
        for coverage, _, tag
        in coverage_spec_629as
    ]
)


# =============================================================================
# 5) Compute frozen corrupted absolute-error evidence ONCE
# =============================================================================

FD003_CORRUPTION_ALL_ABS_ERROR_629AS = np.abs(
    pred_all_629as
    -
    target_629as.reshape(
        1,
        -1,
    )
)

if FD003_CORRUPTION_ALL_ABS_ERROR_629AS.shape != (336, 4706):
    raise RuntimeError(
        "Corrupted absolute-error matrix shape changed."
    )

if not np.isfinite(
    FD003_CORRUPTION_ALL_ABS_ERROR_629AS
).all():
    raise RuntimeError(
        "Corrupted absolute-error matrix contains non-finite values."
    )

FD003_CORRUPTION_ALL_ABS_ERROR_SHA256_629AS = _sha_arr_629as(
    FD003_CORRUPTION_ALL_ABS_ERROR_629AS
)


# =============================================================================
# 6) Scenario-wise scoring
# =============================================================================

manifest_629as = (
    FD003_CORRUPTION_MANIFEST_629AF
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

scenario_rows_629as = []

clean_0p8_629as = clean_selective_629as[
    "0p8"
]

clean_mean_risk_629as = float(
    np.mean(
        clean_risk_629as
    )
)

for i in range(336):

    row = manifest_629as.iloc[
        i
    ]

    err = FD003_CORRUPTION_ALL_ABS_ERROR_629AS[
        i
    ]

    pred = pred_all_629as[
        i
    ]

    mc_std = mc_all_629as[
        i
    ]

    risk = risk_all_629as[
        i
    ]

    full_mae = float(
        np.mean(err)
    )

    median_abs = float(
        np.median(err)
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (
                    pred
                    -
                    target_629as
                ) ** 2
            )
        )
    )

    spearman = _spearman_629as(
        risk,
        err,
    )

    metrics_by_tag = {}

    for coverage, count, tag in coverage_spec_629as:

        metrics_by_tag[
            tag
        ] = _selective_metrics_629as(
            err,
            risk,
            count,
        )

    m08 = metrics_by_tag[
        "0p8"
    ]

    scenario_record = {
        "scenario_number":
            int(row["scenario_number"]),

        "scenario_id":
            str(row["scenario_id"]),

        "family":
            str(row["family"]),

        "sensor":
            str(row["sensor"]),

        "severity":
            str(row["severity"]),

        "alpha":
            (
                np.nan
                if pd.isna(row["alpha"])
                else float(row["alpha"])
            ),

        "direction":
            (
                np.nan
                if pd.isna(row["direction"])
                else float(row["direction"])
            ),

        "requested_fraction":
            (
                np.nan
                if pd.isna(row["requested_fraction"])
                else float(row["requested_fraction"])
            ),

        "seed":
            (
                np.nan
                if pd.isna(row["seed"])
                else float(row["seed"])
            ),

        "full_mae":
            full_mae,

        "median_absolute_error":
            median_abs,

        "rmse":
            rmse,

        "spearman_risk_abs_error":
            spearman,

        "mean_absolute_prediction_shift_vs_clean":
            float(
                np.mean(
                    np.abs(
                        pred
                        -
                        clean_pred_629as
                    )
                )
            ),

        "mean_mc_std":
            float(
                np.mean(mc_std)
            ),

        "mean_final_selective_risk":
            float(
                np.mean(risk)
            ),

        "mean_final_selective_risk_shift_vs_clean":
            float(
                np.mean(risk)
                -
                clean_mean_risk_629as
            ),

        "delta_full_mae_vs_clean":
            float(
                full_mae
                -
                clean_full_mae_629as
            ),

        "ratio_full_mae_to_clean":
            float(
                full_mae
                /
                clean_full_mae_629as
            ),

        "delta_spearman_vs_clean":
            float(
                spearman
                -
                clean_spearman_629as
            ),
    }

    for coverage, _, tag in coverage_spec_629as:

        m = metrics_by_tag[
            tag
        ]

        scenario_record[
            f"accepted_count_{tag}"
        ] = int(
            m["accepted_count"]
        )

        scenario_record[
            f"realized_coverage_{tag}"
        ] = float(
            m["realized_coverage"]
        )

        scenario_record[
            f"accepted_mae_{tag}"
        ] = float(
            m["accepted_mae"]
        )

        scenario_record[
            f"abstained_mae_{tag}"
        ] = float(
            m["abstained_mae"]
        )

        scenario_record[
            f"selective_mae_gain_{tag}"
        ] = float(
            m["selective_mae_gain"]
        )

        scenario_record[
            f"relative_mae_reduction_pct_{tag}"
        ] = float(
            m[
                "relative_mae_reduction_pct"
            ]
        )

        scenario_record[
            f"abstained_minus_accepted_mae_{tag}"
        ] = float(
            m[
                "abstained_minus_accepted_mae"
            ]
        )

    scenario_record[
        "delta_accepted_mae_vs_clean_at_0p8"
    ] = float(
        m08["accepted_mae"]
        -
        clean_0p8_629as["accepted_mae"]
    )

    scenario_record[
        "delta_selective_mae_gain_vs_clean_at_0p8"
    ] = float(
        m08["selective_mae_gain"]
        -
        clean_0p8_629as["selective_mae_gain"]
    )

    scenario_rows_629as.append(
        scenario_record
    )


FD003_CORRUPTION_SCENARIO_METRICS_629AS = pd.DataFrame(
    scenario_rows_629as
)

if len(FD003_CORRUPTION_SCENARIO_METRICS_629AS) != 336:
    raise RuntimeError(
        "Scenario-level scoring table does not contain 336 rows."
    )


# =============================================================================
# 7) Predeclared scenario-level macro summaries
# =============================================================================

macro_metrics_629as = [
    "full_mae",
    "spearman_risk_abs_error",
    "accepted_mae_0p8",
    "selective_mae_gain_0p8",
    "relative_mae_reduction_pct_0p8",
    "delta_full_mae_vs_clean",
    "delta_spearman_vs_clean",
    "delta_accepted_mae_vs_clean_at_0p8",
    "delta_selective_mae_gain_vs_clean_at_0p8",
    "mean_absolute_prediction_shift_vs_clean",
    "mean_mc_std",
    "mean_final_selective_risk",
    "mean_final_selective_risk_shift_vs_clean",
]


def _macro_summary_629as(
    df,
    group_columns,
):

    grouped = df.groupby(
        group_columns,
        sort=True,
        dropna=False,
    )

    summary = grouped[
        macro_metrics_629as
    ].agg(
        [
            "mean",
            "median",
            "min",
            "max",
        ]
    )

    summary.columns = [
        f"{metric}_{stat}"
        for metric, stat in summary.columns
    ]

    counts = grouped.size().rename(
        "scenario_count"
    )

    valid_spearman = grouped[
        "spearman_risk_abs_error"
    ].count().rename(
        "valid_spearman_count"
    )

    return pd.concat(
        [
            counts,
            valid_spearman,
            summary,
        ],
        axis=1,
    ).reset_index()


FD003_CORRUPTION_FAMILY_SUMMARY_629AS = _macro_summary_629as(
    FD003_CORRUPTION_SCENARIO_METRICS_629AS,
    [
        "family",
    ],
)

FD003_CORRUPTION_FAMILY_SEVERITY_SUMMARY_629AS = _macro_summary_629as(
    FD003_CORRUPTION_SCENARIO_METRICS_629AS,
    [
        "family",
        "severity",
    ],
)

FD003_CORRUPTION_SENSOR_SUMMARY_629AS = _macro_summary_629as(
    FD003_CORRUPTION_SCENARIO_METRICS_629AS,
    [
        "sensor",
    ],
)

FD003_CORRUPTION_FAMILY_SENSOR_SUMMARY_629AS = _macro_summary_629as(
    FD003_CORRUPTION_SCENARIO_METRICS_629AS,
    [
        "family",
        "sensor",
    ],
)


# =============================================================================
# 8) Fingerprints
# =============================================================================

FD003_CORRUPTION_SCENARIO_METRICS_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_SCENARIO_METRICS_629AS
)

FD003_CORRUPTION_CLEAN_REFERENCE_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_SCORING_CLEAN_REFERENCE_629AS
)

FD003_CORRUPTION_FAMILY_SUMMARY_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_FAMILY_SUMMARY_629AS
)

FD003_CORRUPTION_FAMILY_SEVERITY_SUMMARY_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_FAMILY_SEVERITY_SUMMARY_629AS
)

FD003_CORRUPTION_SENSOR_SUMMARY_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_SENSOR_SUMMARY_629AS
)

FD003_CORRUPTION_FAMILY_SENSOR_SUMMARY_SHA256_629AS = _df_sha256_629as(
    FD003_CORRUPTION_FAMILY_SENSOR_SUMMARY_629AS
)


# =============================================================================
# 9) Outcome-evidence contract
# =============================================================================

FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS = {

    "step":
        "6.29AS",

    "dataset":
        "FD003",

    "scoring_protocol_sha256":
        FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ,

    "dev_target_authority_contract_sha256":
        FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR,

    "full_execution_contract_sha256":
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP,

    "scenario_count":
        336,

    "windows_per_scenario":
        4706,

    "absolute_error_matrix_sha256":
        FD003_CORRUPTION_ALL_ABS_ERROR_SHA256_629AS,

    "scenario_metrics_sha256":
        FD003_CORRUPTION_SCENARIO_METRICS_SHA256_629AS,

    "clean_reference_sha256":
        FD003_CORRUPTION_CLEAN_REFERENCE_SHA256_629AS,

    "family_summary_sha256":
        FD003_CORRUPTION_FAMILY_SUMMARY_SHA256_629AS,

    "family_severity_summary_sha256":
        FD003_CORRUPTION_FAMILY_SEVERITY_SUMMARY_SHA256_629AS,

    "sensor_summary_sha256":
        FD003_CORRUPTION_SENSOR_SUMMARY_SHA256_629AS,

    "family_sensor_summary_sha256":
        FD003_CORRUPTION_FAMILY_SENSOR_SUMMARY_SHA256_629AS,

    "clean_full_mae":
        clean_full_mae_629as,

    "clean_spearman":
        clean_spearman_629as,

    "primary_diagnostic_coverage":
        0.8,

    "dev_targets_used":
        True,

    "corrupted_errors_computed":
        True,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "model_retrained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS = _sha_json_629as(
    FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS
)


# =============================================================================
# 10) Read-only lock + state
# =============================================================================

FD003_CORRUPTION_ALL_ABS_ERROR_629AS.setflags(
    write=False
)

FD003_CORRUPTION_SCORING_COMPLETE_629AS = True
FD003_CORRUPTION_OUTCOME_EVIDENCE_LOCKED_629AS = True

FD003_DEV_TARGETS_USED_629AS = True
FD003_CORRUPTED_ERRORS_COMPUTED_629AS = True
FD003_NEW_CORRUPTION_EXECUTED_629AS = False
FD003_MODEL_INFERENCE_PERFORMED_629AS = False
FD003_MC_EXECUTION_PERFORMED_629AS = False
FD003_MODEL_RETRAINED_629AS = False
FD003_THRESHOLD_SELECTED_629AS = False
FD003_POLICY_SELECTED_629AS = False
FD003_TEST_CONTENTS_READ_629AS = False
FD003_RUL_CONTENTS_READ_629AS = False

all_checks_passed_629as = True


# =============================================================================
# 11) Compact reporting output
# =============================================================================

scenario_metrics_629as = FD003_CORRUPTION_SCENARIO_METRICS_629AS

print("=" * 160)

print(
    "STEP 6.29AS — FD003 FROZEN 336-SCENARIO "
    "DEV CORRUPTION OUTCOME / RELIABILITY SCORING"
)

print("=" * 160)

print("DEV targets used               :", True)
print("Corrupted errors computed      :", True)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Model retrained                :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- CLEAN REFERENCE UNDER FROZEN SCORING RULES ---"
)

print(
    "Clean full DEV MAE             :",
    clean_full_mae_629as,
)

print(
    "Clean Spearman(risk, abs error):",
    clean_spearman_629as,
)

print(
    "Clean accepted MAE @ 0.8       :",
    clean_0p8_629as[
        "accepted_mae"
    ],
)

print(
    "Clean selective gain @ 0.8     :",
    clean_0p8_629as[
        "selective_mae_gain"
    ],
)

print(
    "\n--- 336-SCENARIO PRIMARY EVIDENCE RANGE ---"
)

print(
    "Full MAE min / median / mean / max:",
    (
        float(
            scenario_metrics_629as[
                "full_mae"
            ].min()
        ),
        float(
            scenario_metrics_629as[
                "full_mae"
            ].median()
        ),
        float(
            scenario_metrics_629as[
                "full_mae"
            ].mean()
        ),
        float(
            scenario_metrics_629as[
                "full_mae"
            ].max()
        ),
    ),
)

print(
    "Spearman min / median / mean / max:",
    (
        float(
            scenario_metrics_629as[
                "spearman_risk_abs_error"
            ].min()
        ),
        float(
            scenario_metrics_629as[
                "spearman_risk_abs_error"
            ].median()
        ),
        float(
            scenario_metrics_629as[
                "spearman_risk_abs_error"
            ].mean()
        ),
        float(
            scenario_metrics_629as[
                "spearman_risk_abs_error"
            ].max()
        ),
    ),
)

print(
    "Selective gain @0.8 min/median/mean/max:",
    (
        float(
            scenario_metrics_629as[
                "selective_mae_gain_0p8"
            ].min()
        ),
        float(
            scenario_metrics_629as[
                "selective_mae_gain_0p8"
            ].median()
        ),
        float(
            scenario_metrics_629as[
                "selective_mae_gain_0p8"
            ].mean()
        ),
        float(
            scenario_metrics_629as[
                "selective_mae_gain_0p8"
            ].max()
        ),
    ),
)

print(
    "\n--- FAMILY-LEVEL MACRO SUMMARY "
    "(primary metrics; scenario-level averaging) ---"
)

family_display_columns_629as = [
    "family",
    "scenario_count",
    "full_mae_mean",
    "full_mae_median",
    "spearman_risk_abs_error_mean",
    "spearman_risk_abs_error_median",
    "accepted_mae_0p8_mean",
    "selective_mae_gain_0p8_mean",
    "relative_mae_reduction_pct_0p8_mean",
    "delta_full_mae_vs_clean_mean",
]

print(
    FD003_CORRUPTION_FAMILY_SUMMARY_629AS[
        family_display_columns_629as
    ].to_string(
        index=False
    )
)

print(
    "\nAbsolute-error matrix SHA-256  :",
    FD003_CORRUPTION_ALL_ABS_ERROR_SHA256_629AS,
)

print(
    "Scenario-metrics SHA-256       :",
    FD003_CORRUPTION_SCENARIO_METRICS_SHA256_629AS,
)

print(
    "Outcome-evidence contract SHA-256:",
    FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,
)

print(
    "\nPASS — the prospectively frozen corruption scoring protocol "
    "was executed exactly once against all 336 locked scenarios."
)

print(
    "No threshold or operating policy was selected, and TEST/official RUL "
    "remain unopened. Next: interpret the locked robustness evidence and "
    "prospectively freeze the DEV-only operating-threshold selection rule."
)

In [ ]:
# STEP 6.29AT
# FD003 DEV-ONLY OPERATING-THRESHOLD SELECTION-RULE FREEZE
#
# Freezes the rule BEFORE computing the numerical threshold.
#
# Threshold source: CLEAN DEV final selective risk only.
# Target nominal coverage: 0.8.
# Corruption outcomes are robustness/reporting evidence ONLY and are excluded
# from numerical threshold construction.
#
# NO threshold value is computed in this step.
# NO policy is activated.
# NO new inference / MC / corruption.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629as", False)):
    raise RuntimeError(
        "STEP 6.29AS must pass before STEP 6.29AT."
    )

required_629at = [
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS",
    "FD003_CORRUPTION_SCORING_PROTOCOL_629AQ",
    "FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V",
]

missing_629at = [
    name
    for name in required_629at
    if name not in globals()
]

if missing_629at:
    raise RuntimeError(
        "Missing STEP 6.29AT object(s): "
        + ", ".join(missing_629at)
    )

if not bool(
    globals().get(
        "FD003_CORRUPTION_OUTCOME_EVIDENCE_LOCKED_629AS",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AS outcome evidence is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AS", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AT."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AS", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AT."
    )

if bool(
    globals().get(
        "FD003_THRESHOLD_SELECTION_RULE_LOCKED_629AT",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AT has already been completed."
    )


# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629at(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AT hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629at(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify frozen evidence identities
# =============================================================================

if (
    _sha_json_629at(
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS
    )
    !=
    FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS
):
    raise RuntimeError(
        "Locked corruption outcome evidence changed before threshold-rule freeze."
    )

if (
    _sha_json_629at(
        FD003_CORRUPTION_SCORING_PROTOCOL_629AQ
    )
    !=
    FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ
):
    raise RuntimeError(
        "Locked corruption scoring protocol changed before threshold-rule freeze."
    )

clean_risk_629at = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
)

if clean_risk_629at.shape != (4706,):
    raise RuntimeError(
        "Frozen clean DEV final-risk vector shape changed."
    )

if (
    _sha_arr_629at(
        clean_risk_629at
    )
    !=
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V
):
    raise RuntimeError(
        "Frozen clean DEV final-risk fingerprint changed."
    )

if not np.isfinite(clean_risk_629at).all():
    raise RuntimeError(
        "Frozen clean DEV final-risk vector contains non-finite values."
    )

if not bool(
    (
        (clean_risk_629at >= 0.0)
        &
        (clean_risk_629at <= 1.0)
    ).all()
):
    raise RuntimeError(
        "Frozen clean DEV final-risk vector is outside [0,1]."
    )


# =============================================================================
# 4) Freeze numerical threshold-selection rule
# =============================================================================

nominal_coverage_629at = 0.8

if (
    FD003_CORRUPTION_SCORING_PROTOCOL_629AQ[
        "primary_selective_operating_diagnostic"
    ]
    !=
    "target coverage 0.8; diagnostic only, not an operating threshold"
):
    raise RuntimeError(
        "The previously frozen 0.8 primary selective diagnostic changed."
    )


FD003_THRESHOLD_SELECTION_RULE_629AT = {

    "step":
        "6.29AT",

    "dataset":
        "FD003",

    "policy_form":
        "uncertainty_only_fixed_threshold",

    "risk_source":
        "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",

    "risk_source_contract_sha256":
        FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V,

    "risk_orientation":
        "lower final selective risk = more reliable = accept",

    "threshold_source_population":
        "clean FD003 DEV windows only",

    "threshold_source_window_count":
        4706,

    "nominal_coverage":
        nominal_coverage_629at,

    "threshold_estimator":
        "np.quantile(clean_final_selective_risk_float64, q=0.8, method='linear')",

    "quantile_q":
        nominal_coverage_629at,

    "quantile_method":
        "linear",

    "quantile_input_dtype":
        "float64",

    "acceptance_comparator":
        "<=",

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "tie_handling":
        "no post-hoc tie breaking or threshold adjustment; "
        "all windows satisfying <= threshold are accepted",

    "realized_coverage_rule":
        "report realized clean-DEV coverage after applying <= threshold; "
        "do not modify threshold to force exact nominal coverage",

    "clean_dev_labels_used_for_threshold":
        False,

    "clean_dev_errors_used_for_threshold":
        False,

    "corruption_predictions_used_for_threshold":
        False,

    "corruption_risks_used_for_threshold":
        False,

    "corruption_errors_used_for_threshold":
        False,

    "corruption_outcomes_role":
        "robustness/reporting evidence only; excluded from numerical threshold construction",

    "corruption_outcome_evidence_sha256":
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,

    "model_retraining_allowed":
        False,

    "risk_architecture_modification_allowed":
        False,

    "manifest_modification_allowed":
        False,

    "threshold_value_computed_in_this_step":
        False,

    "threshold_selected":
        False,

    "policy_activated":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_THRESHOLD_SELECTION_RULE_SHA256_629AT = _sha_json_629at(
    FD003_THRESHOLD_SELECTION_RULE_629AT
)


# =============================================================================
# 5) Lock state
# =============================================================================

FD003_THRESHOLD_SELECTION_RULE_LOCKED_629AT = True

FD003_THRESHOLD_VALUE_COMPUTED_629AT = False
FD003_THRESHOLD_SELECTED_629AT = False
FD003_POLICY_SELECTED_629AT = False
FD003_POLICY_ACTIVATED_629AT = False

FD003_NEW_CORRUPTION_EXECUTED_629AT = False
FD003_MODEL_INFERENCE_PERFORMED_629AT = False
FD003_MC_EXECUTION_PERFORMED_629AT = False
FD003_MODEL_RETRAINED_629AT = False

FD003_TEST_CONTENTS_READ_629AT = False
FD003_RUL_CONTENTS_READ_629AT = False

all_checks_passed_629at = True


# =============================================================================
# 6) Output
# =============================================================================

print("=" * 160)

print(
    "STEP 6.29AT — FD003 DEV-ONLY OPERATING-THRESHOLD "
    "SELECTION-RULE FREEZE"
)

print("=" * 160)

print("Threshold value computed       :", False)
print("Threshold selected             :", False)
print("Policy activated               :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Model retrained                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- FROZEN THRESHOLD-SELECTION RULE ---"
)

print(
    "Policy form                    :",
    "uncertainty_only_fixed_threshold",
)

print(
    "Threshold source               :",
    "clean DEV final selective risk only",
)

print(
    "Nominal coverage               :",
    nominal_coverage_629at,
)

print(
    "Estimator                      :",
    "np.quantile(..., q=0.8, method='linear')",
)

print(
    "Comparator                     :",
    "<=",
)

print(
    "Tie handling                   :",
    "accept all risk <= threshold; no post-hoc adjustment",
)

print(
    "DEV labels/errors used         :",
    False,
)

print(
    "Corruption outcomes used       :",
    False,
)

print(
    "Corruption-outcome role        :",
    "robustness/reporting only",
)

print(
    "\nThreshold-rule SHA-256         :",
    FD003_THRESHOLD_SELECTION_RULE_SHA256_629AT,
)

print(
    "\nPASS — the FD003 operating-threshold selection rule is now "
    "prospectively frozen before computing the numerical threshold."
)

print(
    "Next: apply this frozen rule exactly once to the already-frozen "
    "clean DEV final selective-risk vector, lock the numerical threshold "
    "and realized clean-DEV coverage, and still keep TEST/official RUL unopened."
)

In [ ]:
# STEP 6.29AU
# FD003 NUMERICAL OPERATING-THRESHOLD + CLEAN-COVERAGE LOCK
#
# Applies the already-frozen STEP 6.29AT rule exactly once to the frozen
# clean-DEV final selective-risk vector.
#
# NO DEV labels/errors are used.
# NO corruption outcomes are used.
# NO new inference / MC / corruption.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629at", False)):
    raise RuntimeError(
        "STEP 6.29AT must pass before STEP 6.29AU."
    )

required_629au = [
    "FD003_THRESHOLD_SELECTION_RULE_629AT",
    "FD003_THRESHOLD_SELECTION_RULE_SHA256_629AT",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V",
    "FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS",
]

missing_629au = [
    name
    for name in required_629au
    if name not in globals()
]

if missing_629au:
    raise RuntimeError(
        "Missing STEP 6.29AU object(s): "
        + ", ".join(missing_629au)
    )

if not bool(
    globals().get(
        "FD003_THRESHOLD_SELECTION_RULE_LOCKED_629AT",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AT threshold-selection rule is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AT", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AU."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AT", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AU."
    )

if bool(
    globals().get(
        "FD003_NUMERICAL_THRESHOLD_LOCKED_629AU",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AU has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629au(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AU hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629au(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify the frozen rule + clean risk identity
# =============================================================================

if (
    _sha_json_629au(
        FD003_THRESHOLD_SELECTION_RULE_629AT
    )
    !=
    FD003_THRESHOLD_SELECTION_RULE_SHA256_629AT
):
    raise RuntimeError(
        "Frozen STEP 6.29AT threshold-selection rule changed."
    )

rule_629au = FD003_THRESHOLD_SELECTION_RULE_629AT

if float(rule_629au["nominal_coverage"]) != 0.8:
    raise RuntimeError(
        "Frozen nominal coverage changed."
    )

if str(rule_629au["quantile_method"]) != "linear":
    raise RuntimeError(
        "Frozen quantile method changed."
    )

if str(rule_629au["acceptance_comparator"]) != "<=":
    raise RuntimeError(
        "Frozen acceptance comparator changed."
    )

clean_risk_raw_629au = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V
)

if clean_risk_raw_629au.shape != (4706,):
    raise RuntimeError(
        "Frozen clean DEV final-risk shape changed."
    )

if (
    _sha_arr_629au(
        clean_risk_raw_629au
    )
    !=
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V
):
    raise RuntimeError(
        "Frozen clean DEV final-risk fingerprint changed."
    )

clean_risk_629au = np.asarray(
    clean_risk_raw_629au,
    dtype=np.float64,
).reshape(-1)

if not np.isfinite(clean_risk_629au).all():
    raise RuntimeError(
        "Frozen clean DEV final-risk contains non-finite values."
    )

if not bool(
    (
        (clean_risk_629au >= 0.0)
        &
        (clean_risk_629au <= 1.0)
    ).all()
):
    raise RuntimeError(
        "Frozen clean DEV final-risk is outside [0,1]."
    )


# =============================================================================
# 4) Apply the prospectively frozen numerical rule exactly once
# =============================================================================

nominal_coverage_629au = 0.8

FD003_FROZEN_RISK_THRESHOLD_629AU = float(
    np.quantile(
        clean_risk_629au,
        q=nominal_coverage_629au,
        method="linear",
    )
)

if not np.isfinite(
    FD003_FROZEN_RISK_THRESHOLD_629AU
):
    raise RuntimeError(
        "Computed threshold is non-finite."
    )

if not (
    0.0
    <=
    FD003_FROZEN_RISK_THRESHOLD_629AU
    <=
    1.0
):
    raise RuntimeError(
        "Computed threshold is outside [0,1]."
    )


FD003_CLEAN_DEV_ACCEPT_MASK_629AU = (
    clean_risk_629au
    <=
    FD003_FROZEN_RISK_THRESHOLD_629AU
)

FD003_CLEAN_DEV_ABSTAIN_MASK_629AU = (
    ~FD003_CLEAN_DEV_ACCEPT_MASK_629AU
)


accepted_count_629au = int(
    np.count_nonzero(
        FD003_CLEAN_DEV_ACCEPT_MASK_629AU
    )
)

abstained_count_629au = int(
    np.count_nonzero(
        FD003_CLEAN_DEV_ABSTAIN_MASK_629AU
    )
)

realized_coverage_629au = float(
    accepted_count_629au
    /
    clean_risk_629au.size
)

nominal_count_ceil_629au = int(
    np.ceil(
        nominal_coverage_629au
        *
        clean_risk_629au.size
    )
)

below_count_629au = int(
    np.count_nonzero(
        clean_risk_629au
        <
        FD003_FROZEN_RISK_THRESHOLD_629AU
    )
)

equal_count_629au = int(
    np.count_nonzero(
        clean_risk_629au
        ==
        FD003_FROZEN_RISK_THRESHOLD_629AU
    )
)

above_count_629au = int(
    np.count_nonzero(
        clean_risk_629au
        >
        FD003_FROZEN_RISK_THRESHOLD_629AU
    )
)

if (
    below_count_629au
    +
    equal_count_629au
    +
    above_count_629au
    !=
    4706
):
    raise RuntimeError(
        "Threshold partition does not cover all clean DEV windows."
    )

if accepted_count_629au != (
    below_count_629au
    +
    equal_count_629au
):
    raise RuntimeError(
        "Acceptance count is inconsistent with the frozen <= comparator."
    )

if accepted_count_629au < nominal_count_ceil_629au:
    raise RuntimeError(
        "Realized clean coverage fell below the frozen nominal 0.8 target."
    )


# =============================================================================
# 5) Freeze threshold/mask fingerprints
# =============================================================================

FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU = _sha_arr_629au(
    FD003_CLEAN_DEV_ACCEPT_MASK_629AU
)

FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU = _sha_arr_629au(
    FD003_CLEAN_DEV_ABSTAIN_MASK_629AU
)

FD003_FROZEN_RISK_THRESHOLD_SHA256_629AU = _sha_json_629au(
    {
        "threshold_float64":
            FD003_FROZEN_RISK_THRESHOLD_629AU,
    }
)


# =============================================================================
# 6) Freeze completed operating-policy contract
# =============================================================================

FD003_OPERATING_POLICY_CONTRACT_629AU = {

    "step":
        "6.29AU",

    "dataset":
        "FD003",

    "threshold_selection_rule_sha256":
        FD003_THRESHOLD_SELECTION_RULE_SHA256_629AT,

    "clean_selective_risk_contract_sha256":
        FD003_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_629V,

    "clean_final_risk_sha256":
        FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_629V,

    "corruption_outcome_evidence_sha256":
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,

    "policy_form":
        "uncertainty_only_fixed_threshold",

    "risk_orientation":
        "lower final selective risk = more reliable = accept",

    "threshold_source":
        "clean FD003 DEV final selective risk only",

    "nominal_coverage":
        nominal_coverage_629au,

    "quantile_method":
        "linear",

    "quantile_input_dtype":
        "float64",

    "frozen_risk_threshold":
        FD003_FROZEN_RISK_THRESHOLD_629AU,

    "frozen_risk_threshold_sha256":
        FD003_FROZEN_RISK_THRESHOLD_SHA256_629AU,

    "acceptance_comparator":
        "<=",

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "tie_handling":
        "accept all exact threshold ties; no post-hoc adjustment",

    "clean_dev_window_count":
        4706,

    "nominal_ceil_count":
        nominal_count_ceil_629au,

    "clean_dev_below_threshold_count":
        below_count_629au,

    "clean_dev_equal_threshold_count":
        equal_count_629au,

    "clean_dev_above_threshold_count":
        above_count_629au,

    "clean_dev_accepted_count":
        accepted_count_629au,

    "clean_dev_abstained_count":
        abstained_count_629au,

    "clean_dev_realized_coverage":
        realized_coverage_629au,

    "clean_dev_accept_mask_sha256":
        FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU,

    "clean_dev_abstain_mask_sha256":
        FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU,

    "dev_labels_used_for_threshold":
        False,

    "dev_errors_used_for_threshold":
        False,

    "corruption_predictions_used_for_threshold":
        False,

    "corruption_risks_used_for_threshold":
        False,

    "corruption_errors_used_for_threshold":
        False,

    "model_retrained":
        False,

    "risk_architecture_modified":
        False,

    "threshold_selected":
        True,

    "operating_policy_frozen":
        True,

    "test_policy_activated":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU = _sha_json_629au(
    FD003_OPERATING_POLICY_CONTRACT_629AU
)


# =============================================================================
# 7) Read-only lock + state
# =============================================================================

FD003_CLEAN_DEV_ACCEPT_MASK_629AU.setflags(
    write=False
)

FD003_CLEAN_DEV_ABSTAIN_MASK_629AU.setflags(
    write=False
)

FD003_NUMERICAL_THRESHOLD_LOCKED_629AU = True
FD003_OPERATING_POLICY_LOCKED_629AU = True

FD003_THRESHOLD_VALUE_COMPUTED_629AU = True
FD003_THRESHOLD_SELECTED_629AU = True
FD003_POLICY_SELECTED_629AU = True
FD003_POLICY_ACTIVATED_ON_TEST_629AU = False

FD003_DEV_LABELS_USED_FOR_THRESHOLD_629AU = False
FD003_DEV_ERRORS_USED_FOR_THRESHOLD_629AU = False
FD003_CORRUPTION_OUTCOMES_USED_FOR_THRESHOLD_629AU = False

FD003_NEW_CORRUPTION_EXECUTED_629AU = False
FD003_MODEL_INFERENCE_PERFORMED_629AU = False
FD003_MC_EXECUTION_PERFORMED_629AU = False
FD003_MODEL_RETRAINED_629AU = False

FD003_TEST_CONTENTS_READ_629AU = False
FD003_RUL_CONTENTS_READ_629AU = False

all_checks_passed_629au = True


# =============================================================================
# 8) Output
# =============================================================================

print("=" * 160)

print(
    "STEP 6.29AU — FD003 NUMERICAL OPERATING-THRESHOLD "
    "+ CLEAN-COVERAGE LOCK"
)

print("=" * 160)

print("Threshold value computed       :", True)
print("Threshold selected             :", True)
print("Operating policy frozen        :", True)
print("Policy activated on TEST       :", False)
print("DEV labels used for threshold  :", False)
print("DEV errors used for threshold  :", False)
print("Corruption outcomes used       :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Model retrained                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FROZEN NUMERICAL POLICY ---")

print(
    "Policy form                    :",
    "uncertainty_only_fixed_threshold",
)

print(
    "Nominal coverage               :",
    nominal_coverage_629au,
)

print(
    "Frozen risk threshold          :",
    FD003_FROZEN_RISK_THRESHOLD_629AU,
)

print(
    "Comparator                     :",
    "<=",
)

print(
    "Nominal ceil count             :",
    nominal_count_ceil_629au,
)

print(
    "Below / equal / above threshold:",
    (
        below_count_629au,
        equal_count_629au,
        above_count_629au,
    ),
)

print(
    "Accepted / abstained count     :",
    (
        accepted_count_629au,
        abstained_count_629au,
    ),
)

print(
    "Realized clean DEV coverage    :",
    realized_coverage_629au,
)

print(
    "Accept-mask SHA-256            :",
    FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU,
)

print(
    "Abstain-mask SHA-256           :",
    FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU,
)

print(
    "Threshold SHA-256              :",
    FD003_FROZEN_RISK_THRESHOLD_SHA256_629AU,
)

print(
    "Operating-policy SHA-256       :",
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,
)

print(
    "\nPASS — the prospectively frozen clean-DEV rule was applied exactly "
    "once and the FD003 numerical operating threshold is now locked."
)

print(
    "TEST and official RUL remain unopened. Next: evaluate/report the "
    "already-frozen policy on clean DEV and corruption evidence without "
    "changing it, then prepare the one-way TEST admission contract."
)

In [ ]:
# STEP 6.29AV
# FD003 FROZEN-POLICY DEV REPORTING / ROBUSTNESS-EVALUATION PROTOCOL FREEZE
#
# Freezes how the already-selected STEP 6.29AU operating policy will be
# reported on:
#   1) clean DEV, and
#   2) all 336 already-scored corruption scenarios.
#
# NO policy metric is computed in this step.
# NO threshold is recomputed or modified.
# NO model inference / MC / corruption / retraining.
# NO TEST / official RUL access.

import hashlib
import json


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629au", False)):
    raise RuntimeError(
        "STEP 6.29AU must pass before STEP 6.29AV."
    )

required_629av = [
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
    "FD003_FROZEN_RISK_THRESHOLD_629AU",
    "FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU",
    "FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS",
    "FD003_CORRUPTION_SCORING_PROTOCOL_629AQ",
    "FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ",
]

missing_629av = [
    name
    for name in required_629av
    if name not in globals()
]

if missing_629av:
    raise RuntimeError(
        "Missing STEP 6.29AV object(s): "
        + ", ".join(missing_629av)
    )

if not bool(
    globals().get(
        "FD003_OPERATING_POLICY_LOCKED_629AU",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AU operating policy is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AU", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AV."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AU", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AV."
    )

if bool(
    globals().get(
        "FD003_POLICY_EVALUATION_PROTOCOL_LOCKED_629AV",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AV has already been completed."
    )


# =============================================================================
# 2) Canonical helper
# =============================================================================

def _sha_json_629av(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify locked policy/evidence identities
# =============================================================================

if (
    _sha_json_629av(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    !=
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "Frozen STEP 6.29AU operating-policy contract changed."
    )

if (
    _sha_json_629av(
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS
    )
    !=
    FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS
):
    raise RuntimeError(
        "Locked STEP 6.29AS corruption outcome evidence changed."
    )

if (
    _sha_json_629av(
        FD003_CORRUPTION_SCORING_PROTOCOL_629AQ
    )
    !=
    FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ
):
    raise RuntimeError(
        "Frozen STEP 6.29AQ corruption scoring protocol changed."
    )

policy_629av = FD003_OPERATING_POLICY_CONTRACT_629AU

if str(policy_629av["policy_form"]) != "uncertainty_only_fixed_threshold":
    raise RuntimeError(
        "Frozen operating-policy form changed."
    )

if str(policy_629av["acceptance_comparator"]) != "<=":
    raise RuntimeError(
        "Frozen operating-policy comparator changed."
    )

if float(policy_629av["frozen_risk_threshold"]) != float(
    FD003_FROZEN_RISK_THRESHOLD_629AU
):
    raise RuntimeError(
        "Frozen numerical threshold identity changed."
    )


# =============================================================================
# 4) Prospectively freeze fixed-policy reporting design
# =============================================================================

FD003_POLICY_EVALUATION_PROTOCOL_629AV = {

    "step":
        "6.29AV",

    "dataset":
        "FD003",

    "purpose":
        "report the already-frozen fixed-threshold policy on clean DEV "
        "and previously locked corruption evidence without changing policy",

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "corruption_outcome_evidence_sha256":
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,

    "corruption_scoring_protocol_sha256":
        FD003_CORRUPTION_SCORING_PROTOCOL_SHA256_629AQ,

    "policy_form":
        "uncertainty_only_fixed_threshold",

    "frozen_risk_threshold":
        float(
            FD003_FROZEN_RISK_THRESHOLD_629AU
        ),

    "acceptance_comparator":
        "<=",

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "threshold_recalibration_per_corruption":
        False,

    "threshold_modification_after_reporting":
        False,

    "clean_dev_population":
        4706,

    "corruption_scenario_count":
        336,

    "corruption_windows_per_scenario":
        4706,

    "clean_policy_metrics":
        [
            "accepted_count",
            "abstained_count",
            "realized_coverage",
            "full_MAE",
            "accepted_MAE",
            "abstained_MAE",
            "selective_MAE_gain = full_MAE - accepted_MAE",
            "relative_MAE_reduction_pct = 100 * selective_MAE_gain / full_MAE",
            "abstained_minus_accepted_MAE",
        ],

    "per_corruption_scenario_policy_metrics":
        [
            "accepted_count",
            "abstained_count",
            "realized_coverage",
            "coverage_delta_vs_nominal_0p8",
            "full_MAE",
            "accepted_MAE",
            "abstained_MAE",
            "selective_MAE_gain = full_MAE - accepted_MAE",
            "relative_MAE_reduction_pct = 100 * selective_MAE_gain / full_MAE",
            "abstained_minus_accepted_MAE",
            "delta_accepted_MAE_vs_clean_fixed_policy",
            "delta_selective_MAE_gain_vs_clean_fixed_policy",
        ],

    "empty_subset_rule":
        "if accepted_count == 0, accepted_MAE/selective-gain-derived metrics are NaN; "
        "if abstained_count == 0, abstained_MAE and abstained-minus-accepted_MAE are NaN",

    "primary_fixed_policy_corruption_evidence":
        [
            "realized_coverage distribution across 336 scenarios",
            "accepted_MAE distribution across 336 scenarios",
            "selective_MAE_gain distribution across 336 scenarios",
            "fraction of scenarios with selective_MAE_gain > 0",
        ],

    "predeclared_group_summaries":
        [
            "family",
            "family + severity",
            "sensor",
            "family + sensor",
        ],

    "group_aggregation_rule":
        "macro across scenarios using arithmetic mean, median, minimum, maximum; "
        "also report valid-count for metrics that can be NaN",

    "policy_success_reporting_rule":
        "report both favorable and unfavorable scenarios; no binary pass/fail criterion",

    "clean_policy_mask_identity":
        {
            "accept_mask_sha256":
                FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU,

            "abstain_mask_sha256":
                FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU,
        },

    "corruption_outcomes_may_change_threshold":
        False,

    "corruption_outcomes_may_change_policy":
        False,

    "corruption_outcomes_may_change_risk_architecture":
        False,

    "model_retraining_allowed":
        False,

    "new_model_inference_allowed":
        False,

    "new_mc_execution_allowed":
        False,

    "new_corruption_execution_allowed":
        False,

    "policy_metrics_computed_in_this_step":
        False,

    "threshold_recomputed_in_this_step":
        False,

    "test_admission_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_POLICY_EVALUATION_PROTOCOL_SHA256_629AV = _sha_json_629av(
    FD003_POLICY_EVALUATION_PROTOCOL_629AV
)


# =============================================================================
# 5) Lock state
# =============================================================================

FD003_POLICY_EVALUATION_PROTOCOL_LOCKED_629AV = True

FD003_POLICY_METRICS_COMPUTED_629AV = False
FD003_THRESHOLD_RECOMPUTED_629AV = False
FD003_THRESHOLD_MODIFIED_629AV = False

FD003_NEW_CORRUPTION_EXECUTED_629AV = False
FD003_MODEL_INFERENCE_PERFORMED_629AV = False
FD003_MC_EXECUTION_PERFORMED_629AV = False
FD003_MODEL_RETRAINED_629AV = False

FD003_TEST_ADMISSION_PERFORMED_629AV = False
FD003_TEST_CONTENTS_READ_629AV = False
FD003_RUL_CONTENTS_READ_629AV = False

all_checks_passed_629av = True


# =============================================================================
# 6) Output
# =============================================================================

print("=" * 162)

print(
    "STEP 6.29AV — FD003 FROZEN-POLICY DEV REPORTING / "
    "ROBUSTNESS-EVALUATION PROTOCOL FREEZE"
)

print("=" * 162)

print("Policy metrics computed        :", False)
print("Threshold recomputed           :", False)
print("Threshold modified             :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Model retrained                :", False)
print("TEST admission performed       :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FROZEN REPORTING DESIGN ---")

print(
    "Policy form                    :",
    "uncertainty_only_fixed_threshold",
)

print(
    "Frozen risk threshold          :",
    FD003_FROZEN_RISK_THRESHOLD_629AU,
)

print(
    "Comparator                     :",
    "<=",
)

print(
    "Clean DEV population           :",
    4706,
)

print(
    "Corruption scenarios           :",
    336,
)

print(
    "Threshold recalibration        :",
    False,
)

print(
    "Primary corruption reporting   : "
    "coverage + accepted MAE + selective gain + positive-gain rate"
)

print(
    "Group summaries                : "
    "family, family+severity, sensor, family+sensor"
)

print(
    "Policy-evaluation protocol SHA-256:",
    FD003_POLICY_EVALUATION_PROTOCOL_SHA256_629AV,
)

print(
    "\nPASS — reporting of the already-frozen FD003 operating policy is now "
    "prospectively specified before computing fixed-threshold clean/corruption metrics."
)

print(
    "Next: apply the unchanged threshold to clean DEV and all 336 locked "
    "corruption-risk vectors, lock the reporting evidence, then prepare the "
    "one-way TEST admission contract."
)

In [ ]:
# STEP 6.29AW
# FD003 FROZEN FIXED-THRESHOLD CLEAN + CORRUPTION POLICY REPORTING
#
# Applies the already-frozen STEP 6.29AU threshold unchanged to:
#   1) clean DEV final selective risk
#   2) all 336 locked corruption final-risk vectors
#
# Uses already-computed DEV outcome evidence only for reporting.
#
# NO threshold recalibration/modification.
# NO model retraining.
# NO new corruption / inference / MC execution.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629av", False)):
    raise RuntimeError(
        "STEP 6.29AV must pass before STEP 6.29AW."
    )

required_629aw = [
    "FD003_POLICY_EVALUATION_PROTOCOL_629AV",
    "FD003_POLICY_EVALUATION_PROTOCOL_SHA256_629AV",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
    "FD003_FROZEN_RISK_THRESHOLD_629AU",
    "FD003_CLEAN_DEV_ACCEPT_MASK_629AU",
    "FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU",
    "FD003_CLEAN_DEV_ABSTAIN_MASK_629AU",
    "FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU",
    "FD003_DEV_TARGET_VECTOR_629AR",
    "FD003_CLEAN_DEV_PREDICTION_629L",
    "FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V",
    "FD003_CORRUPTION_ALL_PREDICTIONS_629AP",
    "FD003_CORRUPTION_ALL_FINAL_RISK_629AP",
    "FD003_CORRUPTION_ALL_ABS_ERROR_629AS",
    "FD003_CORRUPTION_ALL_ABS_ERROR_SHA256_629AS",
    "FD003_CORRUPTION_MANIFEST_629AF",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS",
]

missing_629aw = [
    name
    for name in required_629aw
    if name not in globals()
]

if missing_629aw:
    raise RuntimeError(
        "Missing STEP 6.29AW object(s): "
        + ", ".join(missing_629aw)
    )

if not bool(
    globals().get(
        "FD003_POLICY_EVALUATION_PROTOCOL_LOCKED_629AV",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AV policy-evaluation protocol is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AV", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AW."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AV", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AW."
    )

if bool(
    globals().get(
        "FD003_FIXED_POLICY_REPORTING_COMPLETE_629AW",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AW has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629aw(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29AW hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _json_scalar_629aw(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        "Unsupported canonical scalar type: "
        + type(value).__name__
    )


def _sha_json_629aw(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _df_sha256_629aw(df):

    records = []

    for row in df.itertuples(
        index=False,
        name=None,
    ):
        records.append(
            [
                _json_scalar_629aw(x)
                for x in row
            ]
        )

    return _sha_json_629aw(
        {
            "columns": [
                str(c)
                for c in df.columns
            ],
            "records":
                records,
        }
    )


def _fixed_policy_metrics_629aw(
    abs_error,
    risk,
    threshold,
):

    err = np.asarray(
        abs_error,
        dtype=np.float64,
    ).reshape(-1)

    r = np.asarray(
        risk,
        dtype=np.float64,
    ).reshape(-1)

    if err.shape != (4706,):
        raise RuntimeError(
            "Fixed-policy error vector shape changed."
        )

    if r.shape != (4706,):
        raise RuntimeError(
            "Fixed-policy risk vector shape changed."
        )

    accept = (
        r
        <=
        float(threshold)
    )

    abstain = ~accept

    accepted_count = int(
        np.count_nonzero(
            accept
        )
    )

    abstained_count = int(
        np.count_nonzero(
            abstain
        )
    )

    full_mae = float(
        np.mean(err)
    )

    accepted_mae = (
        float(
            np.mean(
                err[
                    accept
                ]
            )
        )
        if accepted_count > 0
        else float("nan")
    )

    abstained_mae = (
        float(
            np.mean(
                err[
                    abstain
                ]
            )
        )
        if abstained_count > 0
        else float("nan")
    )

    selective_gain = (
        float(
            full_mae
            -
            accepted_mae
        )
        if accepted_count > 0
        else float("nan")
    )

    relative_reduction = (
        float(
            100.0
            *
            selective_gain
            /
            full_mae
        )
        if (
            accepted_count > 0
            and
            full_mae != 0.0
        )
        else float("nan")
    )

    abstained_minus_accepted = (
        float(
            abstained_mae
            -
            accepted_mae
        )
        if (
            accepted_count > 0
            and
            abstained_count > 0
        )
        else float("nan")
    )

    return {
        "accept_mask":
            accept,

        "accepted_count":
            accepted_count,

        "abstained_count":
            abstained_count,

        "realized_coverage":
            float(
                accepted_count
                /
                err.size
            ),

        "coverage_delta_vs_nominal_0p8":
            float(
                accepted_count
                /
                err.size
                -
                0.8
            ),

        "full_mae":
            full_mae,

        "accepted_mae":
            accepted_mae,

        "abstained_mae":
            abstained_mae,

        "selective_mae_gain":
            selective_gain,

        "relative_mae_reduction_pct":
            relative_reduction,

        "abstained_minus_accepted_mae":
            abstained_minus_accepted,
    }


# =============================================================================
# 3) Re-certify frozen reporting/policy/evidence contracts
# =============================================================================

if (
    _sha_json_629aw(
        FD003_POLICY_EVALUATION_PROTOCOL_629AV
    )
    !=
    FD003_POLICY_EVALUATION_PROTOCOL_SHA256_629AV
):
    raise RuntimeError(
        "Frozen STEP 6.29AV reporting protocol changed."
    )

if (
    _sha_json_629aw(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    !=
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "Frozen STEP 6.29AU operating policy changed."
    )

if (
    _sha_json_629aw(
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_629AS
    )
    !=
    FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS
):
    raise RuntimeError(
        "Locked STEP 6.29AS corruption outcome evidence changed."
    )

if (
    _sha_arr_629aw(
        FD003_CORRUPTION_ALL_ABS_ERROR_629AS
    )
    !=
    FD003_CORRUPTION_ALL_ABS_ERROR_SHA256_629AS
):
    raise RuntimeError(
        "Locked corruption absolute-error matrix changed."
    )


# =============================================================================
# 4) Frozen threshold identity
# =============================================================================

threshold_629aw = float(
    FD003_FROZEN_RISK_THRESHOLD_629AU
)

if (
    threshold_629aw
    !=
    float(
        FD003_OPERATING_POLICY_CONTRACT_629AU[
            "frozen_risk_threshold"
        ]
    )
):
    raise RuntimeError(
        "Frozen threshold identity changed."
    )

if (
    str(
        FD003_OPERATING_POLICY_CONTRACT_629AU[
            "acceptance_comparator"
        ]
    )
    !=
    "<="
):
    raise RuntimeError(
        "Frozen comparator changed."
    )


# =============================================================================
# 5) Clean fixed-policy reporting
# =============================================================================

target_629aw = np.asarray(
    FD003_DEV_TARGET_VECTOR_629AR,
    dtype=np.float64,
).reshape(-1)

clean_pred_629aw = np.asarray(
    FD003_CLEAN_DEV_PREDICTION_629L,
    dtype=np.float64,
).reshape(-1)

clean_risk_629aw = np.asarray(
    FD003_CLEAN_DEV_FINAL_SELECTIVE_RISK_629V,
    dtype=np.float64,
).reshape(-1)

if (
    target_629aw.shape != (4706,)
    or
    clean_pred_629aw.shape != (4706,)
    or
    clean_risk_629aw.shape != (4706,)
):
    raise RuntimeError(
        "Clean fixed-policy vectors have unexpected shape."
    )

clean_abs_629aw = np.abs(
    clean_pred_629aw
    -
    target_629aw
)

clean_metrics_629aw = _fixed_policy_metrics_629aw(
    clean_abs_629aw,
    clean_risk_629aw,
    threshold_629aw,
)

if not np.array_equal(
    clean_metrics_629aw[
        "accept_mask"
    ],
    np.asarray(
        FD003_CLEAN_DEV_ACCEPT_MASK_629AU,
        dtype=bool,
    ),
):
    raise RuntimeError(
        "Recomputed clean accept mask differs from STEP 6.29AU."
    )

if (
    _sha_arr_629aw(
        clean_metrics_629aw[
            "accept_mask"
        ]
    )
    !=
    FD003_CLEAN_DEV_ACCEPT_MASK_SHA256_629AU
):
    raise RuntimeError(
        "Clean accept-mask fingerprint mismatch."
    )

if (
    _sha_arr_629aw(
        ~clean_metrics_629aw[
            "accept_mask"
        ]
    )
    !=
    FD003_CLEAN_DEV_ABSTAIN_MASK_SHA256_629AU
):
    raise RuntimeError(
        "Clean abstain-mask fingerprint mismatch."
    )


FD003_FIXED_POLICY_CLEAN_METRICS_629AW = pd.DataFrame(
    [
        {
            "threshold":
                threshold_629aw,

            "accepted_count":
                clean_metrics_629aw[
                    "accepted_count"
                ],

            "abstained_count":
                clean_metrics_629aw[
                    "abstained_count"
                ],

            "realized_coverage":
                clean_metrics_629aw[
                    "realized_coverage"
                ],

            "coverage_delta_vs_nominal_0p8":
                clean_metrics_629aw[
                    "coverage_delta_vs_nominal_0p8"
                ],

            "full_mae":
                clean_metrics_629aw[
                    "full_mae"
                ],

            "accepted_mae":
                clean_metrics_629aw[
                    "accepted_mae"
                ],

            "abstained_mae":
                clean_metrics_629aw[
                    "abstained_mae"
                ],

            "selective_mae_gain":
                clean_metrics_629aw[
                    "selective_mae_gain"
                ],

            "relative_mae_reduction_pct":
                clean_metrics_629aw[
                    "relative_mae_reduction_pct"
                ],

            "abstained_minus_accepted_mae":
                clean_metrics_629aw[
                    "abstained_minus_accepted_mae"
                ],
        }
    ]
)


# =============================================================================
# 6) Apply unchanged threshold to all 336 corruption scenarios
# =============================================================================

risk_all_629aw = np.asarray(
    FD003_CORRUPTION_ALL_FINAL_RISK_629AP,
    dtype=np.float64,
)

error_all_629aw = np.asarray(
    FD003_CORRUPTION_ALL_ABS_ERROR_629AS,
    dtype=np.float64,
)

if risk_all_629aw.shape != (336, 4706):
    raise RuntimeError(
        "Corruption risk matrix shape changed."
    )

if error_all_629aw.shape != (336, 4706):
    raise RuntimeError(
        "Corruption absolute-error matrix shape changed."
    )

manifest_629aw = (
    FD003_CORRUPTION_MANIFEST_629AF
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

scenario_rows_629aw = []

mask_matrix_629aw = np.empty(
    (
        336,
        4706,
    ),
    dtype=bool,
)

for i in range(336):

    row = manifest_629aw.iloc[
        i
    ]

    metrics = _fixed_policy_metrics_629aw(
        error_all_629aw[
            i
        ],
        risk_all_629aw[
            i
        ],
        threshold_629aw,
    )

    mask_matrix_629aw[
        i
    ] = metrics[
        "accept_mask"
    ]

    scenario_rows_629aw.append(
        {
            "scenario_number":
                int(
                    row[
                        "scenario_number"
                    ]
                ),

            "scenario_id":
                str(
                    row[
                        "scenario_id"
                    ]
                ),

            "family":
                str(
                    row[
                        "family"
                    ]
                ),

            "sensor":
                str(
                    row[
                        "sensor"
                    ]
                ),

            "severity":
                str(
                    row[
                        "severity"
                    ]
                ),

            "alpha":
                (
                    np.nan
                    if pd.isna(
                        row[
                            "alpha"
                        ]
                    )
                    else float(
                        row[
                            "alpha"
                        ]
                    )
                ),

            "direction":
                (
                    np.nan
                    if pd.isna(
                        row[
                            "direction"
                        ]
                    )
                    else float(
                        row[
                            "direction"
                        ]
                    )
                ),

            "requested_fraction":
                (
                    np.nan
                    if pd.isna(
                        row[
                            "requested_fraction"
                        ]
                    )
                    else float(
                        row[
                            "requested_fraction"
                        ]
                    )
                ),

            "seed":
                (
                    np.nan
                    if pd.isna(
                        row[
                            "seed"
                        ]
                    )
                    else float(
                        row[
                            "seed"
                        ]
                    )
                ),

            "accepted_count":
                metrics[
                    "accepted_count"
                ],

            "abstained_count":
                metrics[
                    "abstained_count"
                ],

            "realized_coverage":
                metrics[
                    "realized_coverage"
                ],

            "coverage_delta_vs_nominal_0p8":
                metrics[
                    "coverage_delta_vs_nominal_0p8"
                ],

            "full_mae":
                metrics[
                    "full_mae"
                ],

            "accepted_mae":
                metrics[
                    "accepted_mae"
                ],

            "abstained_mae":
                metrics[
                    "abstained_mae"
                ],

            "selective_mae_gain":
                metrics[
                    "selective_mae_gain"
                ],

            "relative_mae_reduction_pct":
                metrics[
                    "relative_mae_reduction_pct"
                ],

            "abstained_minus_accepted_mae":
                metrics[
                    "abstained_minus_accepted_mae"
                ],

            "delta_accepted_mae_vs_clean_fixed_policy":
                float(
                    metrics[
                        "accepted_mae"
                    ]
                    -
                    clean_metrics_629aw[
                        "accepted_mae"
                    ]
                )
                if np.isfinite(
                    metrics[
                        "accepted_mae"
                    ]
                )
                else np.nan,

            "delta_selective_mae_gain_vs_clean_fixed_policy":
                float(
                    metrics[
                        "selective_mae_gain"
                    ]
                    -
                    clean_metrics_629aw[
                        "selective_mae_gain"
                    ]
                )
                if np.isfinite(
                    metrics[
                        "selective_mae_gain"
                    ]
                )
                else np.nan,
        }
    )


FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW = pd.DataFrame(
    scenario_rows_629aw
)

FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_629AW = (
    mask_matrix_629aw
)

if FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW.shape[0] != 336:
    raise RuntimeError(
        "Fixed-policy corruption metrics table does not contain 336 scenarios."
    )

if (
    FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_629AW.shape
    !=
    (336, 4706)
):
    raise RuntimeError(
        "Fixed-policy corruption accept-mask matrix shape changed."
    )


# =============================================================================
# 7) Frozen-policy aggregate evidence
# =============================================================================

positive_gain_mask_629aw = (
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW[
        "selective_mae_gain"
    ]
    >
    0.0
)

zero_gain_mask_629aw = (
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW[
        "selective_mae_gain"
    ]
    ==
    0.0
)

negative_gain_mask_629aw = (
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW[
        "selective_mae_gain"
    ]
    <
    0.0
)

positive_gain_count_629aw = int(
    np.count_nonzero(
        positive_gain_mask_629aw
    )
)

zero_gain_count_629aw = int(
    np.count_nonzero(
        zero_gain_mask_629aw
    )
)

negative_gain_count_629aw = int(
    np.count_nonzero(
        negative_gain_mask_629aw
    )
)

positive_gain_rate_629aw = float(
    positive_gain_count_629aw
    /
    336
)


# =============================================================================
# 8) Predeclared macro group summaries
# =============================================================================

macro_metrics_629aw = [
    "accepted_count",
    "abstained_count",
    "realized_coverage",
    "coverage_delta_vs_nominal_0p8",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_mae_gain",
    "relative_mae_reduction_pct",
    "abstained_minus_accepted_mae",
    "delta_accepted_mae_vs_clean_fixed_policy",
    "delta_selective_mae_gain_vs_clean_fixed_policy",
]


def _macro_summary_629aw(
    df,
    group_columns,
):

    grouped = df.groupby(
        group_columns,
        sort=True,
        dropna=False,
    )

    summary = grouped[
        macro_metrics_629aw
    ].agg(
        [
            "mean",
            "median",
            "min",
            "max",
            "count",
        ]
    )

    summary.columns = [
        f"{metric}_{stat}"
        for metric, stat in summary.columns
    ]

    scenario_count = grouped.size().rename(
        "scenario_count"
    )

    positive_gain_count = grouped[
        "selective_mae_gain"
    ].apply(
        lambda s: int(
            np.count_nonzero(
                np.asarray(
                    s,
                    dtype=np.float64,
                )
                >
                0.0
            )
        )
    ).rename(
        "positive_gain_count"
    )

    result = pd.concat(
        [
            scenario_count,
            positive_gain_count,
            summary,
        ],
        axis=1,
    ).reset_index()

    result[
        "positive_gain_rate"
    ] = (
        result[
            "positive_gain_count"
        ]
        /
        result[
            "scenario_count"
        ]
    )

    return result


FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW = _macro_summary_629aw(
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW,
    [
        "family",
    ],
)

FD003_FIXED_POLICY_FAMILY_SEVERITY_SUMMARY_629AW = _macro_summary_629aw(
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW,
    [
        "family",
        "severity",
    ],
)

FD003_FIXED_POLICY_SENSOR_SUMMARY_629AW = _macro_summary_629aw(
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW,
    [
        "sensor",
    ],
)

FD003_FIXED_POLICY_FAMILY_SENSOR_SUMMARY_629AW = _macro_summary_629aw(
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW,
    [
        "family",
        "sensor",
    ],
)


# =============================================================================
# 9) Fingerprints
# =============================================================================

FD003_FIXED_POLICY_CLEAN_METRICS_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_CLEAN_METRICS_629AW
)

FD003_FIXED_POLICY_CORRUPTION_METRICS_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW
)

FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_SHA256_629AW = _sha_arr_629aw(
    FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_629AW
)

FD003_FIXED_POLICY_FAMILY_SUMMARY_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW
)

FD003_FIXED_POLICY_FAMILY_SEVERITY_SUMMARY_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_FAMILY_SEVERITY_SUMMARY_629AW
)

FD003_FIXED_POLICY_SENSOR_SUMMARY_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_SENSOR_SUMMARY_629AW
)

FD003_FIXED_POLICY_FAMILY_SENSOR_SUMMARY_SHA256_629AW = _df_sha256_629aw(
    FD003_FIXED_POLICY_FAMILY_SENSOR_SUMMARY_629AW
)


# =============================================================================
# 10) Reporting-evidence contract
# =============================================================================

FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW = {

    "step":
        "6.29AW",

    "dataset":
        "FD003",

    "policy_evaluation_protocol_sha256":
        FD003_POLICY_EVALUATION_PROTOCOL_SHA256_629AV,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "corruption_outcome_evidence_sha256":
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,

    "frozen_risk_threshold":
        threshold_629aw,

    "acceptance_comparator":
        "<=",

    "threshold_recomputed":
        False,

    "threshold_modified":
        False,

    "clean_metrics_sha256":
        FD003_FIXED_POLICY_CLEAN_METRICS_SHA256_629AW,

    "corruption_metrics_sha256":
        FD003_FIXED_POLICY_CORRUPTION_METRICS_SHA256_629AW,

    "corruption_accept_mask_sha256":
        FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_SHA256_629AW,

    "family_summary_sha256":
        FD003_FIXED_POLICY_FAMILY_SUMMARY_SHA256_629AW,

    "family_severity_summary_sha256":
        FD003_FIXED_POLICY_FAMILY_SEVERITY_SUMMARY_SHA256_629AW,

    "sensor_summary_sha256":
        FD003_FIXED_POLICY_SENSOR_SUMMARY_SHA256_629AW,

    "family_sensor_summary_sha256":
        FD003_FIXED_POLICY_FAMILY_SENSOR_SUMMARY_SHA256_629AW,

    "clean_accepted_count":
        int(
            clean_metrics_629aw[
                "accepted_count"
            ]
        ),

    "clean_abstained_count":
        int(
            clean_metrics_629aw[
                "abstained_count"
            ]
        ),

    "clean_realized_coverage":
        float(
            clean_metrics_629aw[
                "realized_coverage"
            ]
        ),

    "clean_full_mae":
        float(
            clean_metrics_629aw[
                "full_mae"
            ]
        ),

    "clean_accepted_mae":
        float(
            clean_metrics_629aw[
                "accepted_mae"
            ]
        ),

    "clean_abstained_mae":
        float(
            clean_metrics_629aw[
                "abstained_mae"
            ]
        ),

    "clean_selective_mae_gain":
        float(
            clean_metrics_629aw[
                "selective_mae_gain"
            ]
        ),

    "corruption_scenario_count":
        336,

    "positive_gain_count":
        positive_gain_count_629aw,

    "zero_gain_count":
        zero_gain_count_629aw,

    "negative_gain_count":
        negative_gain_count_629aw,

    "positive_gain_rate":
        positive_gain_rate_629aw,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "model_retrained":
        False,

    "threshold_recomputed_or_modified":
        False,

    "test_admission_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW = _sha_json_629aw(
    FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW
)


# =============================================================================
# 11) Read-only lock + state
# =============================================================================

FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_629AW.setflags(
    write=False
)

FD003_FIXED_POLICY_REPORTING_COMPLETE_629AW = True
FD003_FIXED_POLICY_REPORTING_LOCKED_629AW = True

FD003_THRESHOLD_RECOMPUTED_629AW = False
FD003_THRESHOLD_MODIFIED_629AW = False

FD003_NEW_CORRUPTION_EXECUTED_629AW = False
FD003_MODEL_INFERENCE_PERFORMED_629AW = False
FD003_MC_EXECUTION_PERFORMED_629AW = False
FD003_MODEL_RETRAINED_629AW = False

FD003_TEST_ADMISSION_PERFORMED_629AW = False
FD003_TEST_CONTENTS_READ_629AW = False
FD003_RUL_CONTENTS_READ_629AW = False

all_checks_passed_629aw = True


# =============================================================================
# 12) Output
# =============================================================================

policy_df_629aw = FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW

print("=" * 164)

print(
    "STEP 6.29AW — FD003 FROZEN FIXED-THRESHOLD "
    "CLEAN + CORRUPTION POLICY REPORTING"
)

print("=" * 164)

print("Threshold recomputed           :", False)
print("Threshold modified             :", False)
print("New corruption executed        :", False)
print("New model inference performed  :", False)
print("New MC-dropout execution       :", False)
print("Model retrained                :", False)
print("TEST admission performed       :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- CLEAN DEV FIXED-POLICY EVIDENCE ---"
)

print(
    "Frozen threshold               :",
    threshold_629aw,
)

print(
    "Accepted / abstained           :",
    (
        clean_metrics_629aw[
            "accepted_count"
        ],
        clean_metrics_629aw[
            "abstained_count"
        ],
    ),
)

print(
    "Realized coverage              :",
    clean_metrics_629aw[
        "realized_coverage"
    ],
)

print(
    "Full / accepted / abstained MAE:",
    (
        clean_metrics_629aw[
            "full_mae"
        ],
        clean_metrics_629aw[
            "accepted_mae"
        ],
        clean_metrics_629aw[
            "abstained_mae"
        ],
    ),
)

print(
    "Selective MAE gain             :",
    clean_metrics_629aw[
        "selective_mae_gain"
    ],
)

print(
    "Relative MAE reduction (%)     :",
    clean_metrics_629aw[
        "relative_mae_reduction_pct"
    ],
)

print(
    "Abstained - accepted MAE       :",
    clean_metrics_629aw[
        "abstained_minus_accepted_mae"
    ],
)

print(
    "\n--- 336 CORRUPTION SCENARIOS UNDER THE SAME FIXED THRESHOLD ---"
)

print(
    "Coverage min / median / mean / max:",
    (
        float(
            policy_df_629aw[
                "realized_coverage"
            ].min()
        ),
        float(
            policy_df_629aw[
                "realized_coverage"
            ].median()
        ),
        float(
            policy_df_629aw[
                "realized_coverage"
            ].mean()
        ),
        float(
            policy_df_629aw[
                "realized_coverage"
            ].max()
        ),
    ),
)

print(
    "Accepted MAE min/median/mean/max:",
    (
        float(
            policy_df_629aw[
                "accepted_mae"
            ].min()
        ),
        float(
            policy_df_629aw[
                "accepted_mae"
            ].median()
        ),
        float(
            policy_df_629aw[
                "accepted_mae"
            ].mean()
        ),
        float(
            policy_df_629aw[
                "accepted_mae"
            ].max()
        ),
    ),
)

print(
    "Selective gain min/median/mean/max:",
    (
        float(
            policy_df_629aw[
                "selective_mae_gain"
            ].min()
        ),
        float(
            policy_df_629aw[
                "selective_mae_gain"
            ].median()
        ),
        float(
            policy_df_629aw[
                "selective_mae_gain"
            ].mean()
        ),
        float(
            policy_df_629aw[
                "selective_mae_gain"
            ].max()
        ),
    ),
)

print(
    "Positive / zero / negative gain:",
    (
        positive_gain_count_629aw,
        zero_gain_count_629aw,
        negative_gain_count_629aw,
    ),
)

print(
    "Positive-gain rate             :",
    positive_gain_rate_629aw,
)

print(
    "\n--- FAMILY-LEVEL FIXED-POLICY MACRO SUMMARY ---"
)

family_columns_629aw = [
    "family",
    "scenario_count",
    "realized_coverage_mean",
    "accepted_mae_mean",
    "selective_mae_gain_mean",
    "relative_mae_reduction_pct_mean",
    "positive_gain_count",
    "positive_gain_rate",
]

print(
    FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW[
        family_columns_629aw
    ].to_string(
        index=False
    )
)

print(
    "\nClean metrics SHA-256          :",
    FD003_FIXED_POLICY_CLEAN_METRICS_SHA256_629AW,
)

print(
    "Corruption metrics SHA-256     :",
    FD003_FIXED_POLICY_CORRUPTION_METRICS_SHA256_629AW,
)

print(
    "Corruption accept-mask SHA-256 :",
    FD003_FIXED_POLICY_CORRUPTION_ACCEPT_MASK_SHA256_629AW,
)

print(
    "Reporting contract SHA-256     :",
    FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,
)

print(
    "\nPASS — the unchanged frozen FD003 threshold has now been reported "
    "on clean DEV and all 336 locked corruption scenarios exactly as prospectively specified."
)

print(
    "The threshold/policy was not changed. TEST and official RUL remain unopened. "
    "Next: interpret and lock the DEV-stage conclusion, then create the one-way "
    "TEST admission contract before opening TEST features."
)

In [ ]:
# STEP 6.29AX
# FD003 DEV-STAGE SCIENTIFIC CONCLUSION + PRE-TEST EVIDENCE CLOSURE
#
# Locks a descriptive conclusion from already-frozen clean/corruption evidence.
# Certifies readiness to create the one-way TEST admission contract.
#
# NO threshold/policy changes.
# NO new scoring, corruption, inference, MC, or retraining.
# NO TEST / official RUL access.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629aw", False)):
    raise RuntimeError(
        "STEP 6.29AW must pass before STEP 6.29AX."
    )

required_629ax = [
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW",
    "FD003_FIXED_POLICY_CLEAN_METRICS_629AW",
    "FD003_FIXED_POLICY_CLEAN_METRICS_SHA256_629AW",
    "FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW",
    "FD003_FIXED_POLICY_CORRUPTION_METRICS_SHA256_629AW",
    "FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW",
    "FD003_FIXED_POLICY_FAMILY_SUMMARY_SHA256_629AW",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
    "FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS",
    "FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP",
    "FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR",
    "FD003_FROZEN_RISK_THRESHOLD_629AU",
]

missing_629ax = [
    name for name in required_629ax
    if name not in globals()
]

if missing_629ax:
    raise RuntimeError(
        "Missing STEP 6.29AX object(s): "
        + ", ".join(missing_629ax)
    )

if not bool(
    globals().get(
        "FD003_FIXED_POLICY_REPORTING_LOCKED_629AW",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AW fixed-policy reporting evidence is not locked."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AW", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AX."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AW", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AX."
    )

if bool(
    globals().get(
        "FD003_DEV_STAGE_CLOSED_629AX",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AX has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_json_629ax(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _json_scalar_629ax(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        "Unsupported canonical scalar type: "
        + type(value).__name__
    )


def _df_sha256_629ax(df):

    return _sha_json_629ax(
        {
            "columns": [
                str(c)
                for c in df.columns
            ],
            "records": [
                [
                    _json_scalar_629ax(x)
                    for x in row
                ]
                for row in df.itertuples(
                    index=False,
                    name=None,
                )
            ],
        }
    )


# =============================================================================
# 3) Re-certify locked reporting evidence
# =============================================================================

if (
    _sha_json_629ax(
        FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW
    )
    !=
    FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW
):
    raise RuntimeError(
        "STEP 6.29AW reporting contract changed."
    )

if (
    _df_sha256_629ax(
        FD003_FIXED_POLICY_CLEAN_METRICS_629AW
    )
    !=
    FD003_FIXED_POLICY_CLEAN_METRICS_SHA256_629AW
):
    raise RuntimeError(
        "STEP 6.29AW clean fixed-policy metrics changed."
    )

if (
    _df_sha256_629ax(
        FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW
    )
    !=
    FD003_FIXED_POLICY_CORRUPTION_METRICS_SHA256_629AW
):
    raise RuntimeError(
        "STEP 6.29AW corruption fixed-policy metrics changed."
    )

if (
    _df_sha256_629ax(
        FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW
    )
    !=
    FD003_FIXED_POLICY_FAMILY_SUMMARY_SHA256_629AW
):
    raise RuntimeError(
        "STEP 6.29AW family summary changed."
    )

if (
    _sha_json_629ax(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    !=
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "STEP 6.29AU operating-policy contract changed."
    )


# =============================================================================
# 4) Recover already-locked scientific evidence
# =============================================================================

clean_629ax = FD003_FIXED_POLICY_CLEAN_METRICS_629AW.iloc[0]
corr_629ax = FD003_FIXED_POLICY_CORRUPTION_METRICS_629AW
family_629ax = FD003_FIXED_POLICY_FAMILY_SUMMARY_629AW

if len(corr_629ax) != 336:
    raise RuntimeError(
        "Corruption reporting table no longer contains 336 scenarios."
    )

threshold_629ax = float(
    FD003_FROZEN_RISK_THRESHOLD_629AU
)

clean_coverage_629ax = float(
    clean_629ax["realized_coverage"]
)

clean_full_mae_629ax = float(
    clean_629ax["full_mae"]
)

clean_accepted_mae_629ax = float(
    clean_629ax["accepted_mae"]
)

clean_abstained_mae_629ax = float(
    clean_629ax["abstained_mae"]
)

clean_gain_629ax = float(
    clean_629ax["selective_mae_gain"]
)

clean_relative_reduction_629ax = float(
    clean_629ax["relative_mae_reduction_pct"]
)

coverage_min_629ax = float(
    corr_629ax["realized_coverage"].min()
)

coverage_median_629ax = float(
    corr_629ax["realized_coverage"].median()
)

coverage_mean_629ax = float(
    corr_629ax["realized_coverage"].mean()
)

coverage_max_629ax = float(
    corr_629ax["realized_coverage"].max()
)

gain_min_629ax = float(
    corr_629ax["selective_mae_gain"].min()
)

gain_median_629ax = float(
    corr_629ax["selective_mae_gain"].median()
)

gain_mean_629ax = float(
    corr_629ax["selective_mae_gain"].mean()
)

gain_max_629ax = float(
    corr_629ax["selective_mae_gain"].max()
)

positive_gain_count_629ax = int(
    np.count_nonzero(
        corr_629ax[
            "selective_mae_gain"
        ].to_numpy(dtype=np.float64)
        >
        0.0
    )
)

zero_gain_count_629ax = int(
    np.count_nonzero(
        corr_629ax[
            "selective_mae_gain"
        ].to_numpy(dtype=np.float64)
        ==
        0.0
    )
)

negative_gain_count_629ax = int(
    np.count_nonzero(
        corr_629ax[
            "selective_mae_gain"
        ].to_numpy(dtype=np.float64)
        <
        0.0
    )
)

positive_gain_rate_629ax = float(
    positive_gain_count_629ax
    /
    336
)

negative_cases_629ax = (
    corr_629ax.loc[
        corr_629ax["selective_mae_gain"] < 0.0,
        [
            "scenario_number",
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "realized_coverage",
            "full_mae",
            "accepted_mae",
            "selective_mae_gain",
        ],
    ]
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(negative_cases_629ax) != 3:
    raise RuntimeError(
        "Expected exactly 3 negative-gain corruption scenarios "
        "from locked STEP 6.29AW evidence."
    )

negative_families_629ax = sorted(
    negative_cases_629ax[
        "family"
    ].astype(str).unique().tolist()
)

if negative_families_629ax != ["bias"]:
    raise RuntimeError(
        "Locked negative-gain cases are no longer confined to bias."
    )

family_positive_rates_629ax = {
    str(row["family"]):
        float(row["positive_gain_rate"])
    for _, row in family_629ax.iterrows()
}

if set(family_positive_rates_629ax.keys()) != {
    "bias",
    "drift",
    "dropout",
    "noise",
    "stuck",
}:
    raise RuntimeError(
        "Unexpected family set in fixed-policy summary."
    )


# =============================================================================
# 5) Freeze descriptive DEV-stage scientific conclusion
# =============================================================================

FD003_DEV_STAGE_CONCLUSION_629AX = {

    "step":
        "6.29AX",

    "dataset":
        "FD003",

    "research_problem":
        "Reliability-Aware Selective Remaining Useful Life Prediction "
        "under Sensor Degradation",

    "operating_policy":
        "uncertainty_only_fixed_threshold",

    "frozen_risk_threshold":
        threshold_629ax,

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "clean_dev":
        {
            "realized_coverage":
                clean_coverage_629ax,

            "full_mae":
                clean_full_mae_629ax,

            "accepted_mae":
                clean_accepted_mae_629ax,

            "abstained_mae":
                clean_abstained_mae_629ax,

            "selective_mae_gain":
                clean_gain_629ax,

            "relative_mae_reduction_pct":
                clean_relative_reduction_629ax,
        },

    "controlled_corruption_fixed_policy":
        {
            "scenario_count":
                336,

            "coverage_min":
                coverage_min_629ax,

            "coverage_median":
                coverage_median_629ax,

            "coverage_mean":
                coverage_mean_629ax,

            "coverage_max":
                coverage_max_629ax,

            "selective_gain_min":
                gain_min_629ax,

            "selective_gain_median":
                gain_median_629ax,

            "selective_gain_mean":
                gain_mean_629ax,

            "selective_gain_max":
                gain_max_629ax,

            "positive_gain_count":
                positive_gain_count_629ax,

            "zero_gain_count":
                zero_gain_count_629ax,

            "negative_gain_count":
                negative_gain_count_629ax,

            "positive_gain_rate":
                positive_gain_rate_629ax,

            "negative_gain_families":
                negative_families_629ax,

            "family_positive_gain_rates":
                family_positive_rates_629ax,
        },

    "scientific_interpretation":
        (
            "The frozen reliability-aware selective policy improves clean DEV MAE "
            "on the accepted subset and retains positive selective MAE gain in "
            "333 of 336 controlled corruption scenarios. Coverage remains centered "
            "near the nominal 0.8 operating level without per-corruption threshold "
            "recalibration. Robustness is not universal: three bias scenarios show "
            "negative selective gain and must be reported as unfavorable cases."
        ),

    "claim_boundary":
        (
            "DEV and controlled-corruption evidence supports proceeding to a "
            "one-way held-out TEST evaluation of the already-frozen system; it does "
            "not establish held-out TEST performance before TEST is opened."
        ),

    "policy_changed_after_corruption_reporting":
        False,

    "model_retrained_after_corruption_reporting":
        False,

    "risk_architecture_changed_after_corruption_reporting":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD003_NEGATIVE_GAIN_CASES_629AX = negative_cases_629ax

FD003_NEGATIVE_GAIN_CASES_SHA256_629AX = _df_sha256_629ax(
    FD003_NEGATIVE_GAIN_CASES_629AX
)

FD003_DEV_STAGE_CONCLUSION_629AX[
    "negative_gain_cases_sha256"
] = FD003_NEGATIVE_GAIN_CASES_SHA256_629AX


FD003_DEV_STAGE_CONCLUSION_SHA256_629AX = _sha_json_629ax(
    FD003_DEV_STAGE_CONCLUSION_629AX
)


# =============================================================================
# 6) Freeze pre-TEST methodological closure
# =============================================================================

FD003_PRETEST_EVIDENCE_CLOSURE_629AX = {

    "step":
        "6.29AX",

    "dataset":
        "FD003",

    "dev_stage_conclusion_sha256":
        FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,

    "fixed_policy_reporting_contract_sha256":
        FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "corruption_outcome_evidence_contract_sha256":
        FD003_CORRUPTION_OUTCOME_EVIDENCE_CONTRACT_SHA256_629AS,

    "corruption_execution_contract_sha256":
        FD003_CORRUPTION_FULL_EXECUTION_CONTRACT_SHA256_629AP,

    "dev_target_authority_contract_sha256":
        FD003_DEV_TARGET_AUTHORITY_CONTRACT_SHA256_629AR,

    "all_model_training_complete":
        True,

    "model_retraining_after_dev_closure_allowed":
        False,

    "risk_architecture_frozen":
        True,

    "corruption_manifest_frozen":
        True,

    "corruption_execution_complete":
        True,

    "corruption_reporting_complete":
        True,

    "operating_threshold_frozen":
        True,

    "operating_policy_frozen":
        True,

    "test_admission_contract_created":
        False,

    "test_admission_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "ready_to_create_one_way_test_admission_contract":
        True,

    "readiness_basis":
        (
            "all training, calibration, corruption evaluation, threshold selection, "
            "and DEV-stage reporting artifacts are frozen and fingerprinted before "
            "TEST access"
        ),
}


FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX = _sha_json_629ax(
    FD003_PRETEST_EVIDENCE_CLOSURE_629AX
)


# =============================================================================
# 7) State lock
# =============================================================================

FD003_DEV_STAGE_CLOSED_629AX = True
FD003_PRETEST_EVIDENCE_CLOSED_629AX = True
FD003_READY_FOR_TEST_ADMISSION_CONTRACT_629AX = True

FD003_THRESHOLD_MODIFIED_629AX = False
FD003_POLICY_MODIFIED_629AX = False
FD003_MODEL_RETRAINED_629AX = False
FD003_RISK_ARCHITECTURE_MODIFIED_629AX = False

FD003_TEST_ADMISSION_CONTRACT_CREATED_629AX = False
FD003_TEST_ADMISSION_PERFORMED_629AX = False
FD003_TEST_CONTENTS_READ_629AX = False
FD003_RUL_CONTENTS_READ_629AX = False

all_checks_passed_629ax = True


# =============================================================================
# 8) Output
# =============================================================================

print("=" * 164)

print(
    "STEP 6.29AX — FD003 DEV-STAGE SCIENTIFIC CONCLUSION "
    "+ PRE-TEST EVIDENCE CLOSURE"
)

print("=" * 164)

print("Threshold modified             :", False)
print("Policy modified                :", False)
print("Model retrained                :", False)
print("Risk architecture modified     :", False)
print("TEST admission performed       :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- CLEAN DEV CONCLUSION ---"
)

print(
    "Frozen threshold               :",
    threshold_629ax,
)

print(
    "Realized coverage              :",
    clean_coverage_629ax,
)

print(
    "Full / accepted / abstained MAE:",
    (
        clean_full_mae_629ax,
        clean_accepted_mae_629ax,
        clean_abstained_mae_629ax,
    ),
)

print(
    "Selective MAE gain             :",
    clean_gain_629ax,
)

print(
    "Relative MAE reduction (%)     :",
    clean_relative_reduction_629ax,
)

print(
    "\n--- CONTROLLED-CORRUPTION CONCLUSION ---"
)

print(
    "Coverage min/median/mean/max   :",
    (
        coverage_min_629ax,
        coverage_median_629ax,
        coverage_mean_629ax,
        coverage_max_629ax,
    ),
)

print(
    "Selective gain min/median/mean/max:",
    (
        gain_min_629ax,
        gain_median_629ax,
        gain_mean_629ax,
        gain_max_629ax,
    ),
)

print(
    "Positive / zero / negative gain:",
    (
        positive_gain_count_629ax,
        zero_gain_count_629ax,
        negative_gain_count_629ax,
    ),
)

print(
    "Positive-gain rate             :",
    positive_gain_rate_629ax,
)

print(
    "Negative-gain family/families  :",
    negative_families_629ax,
)

print(
    "\nNegative-gain scenarios:"
)

print(
    FD003_NEGATIVE_GAIN_CASES_629AX.to_string(
        index=False
    )
)

print(
    "\nDEV-stage conclusion SHA-256   :",
    FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,
)

print(
    "Pre-TEST closure SHA-256       :",
    FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,
)

print(
    "\nPASS — the FD003 DEV stage is now scientifically interpreted, "
    "fingerprinted, and closed without changing the frozen model, risk "
    "architecture, threshold, or policy."
)

print(
    "TEST and official RUL remain unopened. Next: prospectively create "
    "the one-way TEST admission contract before reading any TEST feature values."
)

In [ ]:
# STEP 6.29AY
# FD003 ONE-WAY TEST ADMISSION CONTRACT FREEZE — PRE-OPEN
#
# Freezes exactly how held-out TEST may be opened and evaluated.
#
# CRITICAL ORDER:
#   A) Open TEST FEATURES only.
#   B) Produce/fingerprint TEST endpoint predictions, MC uncertainty,
#      final reliability risk, and fixed-threshold accept/abstain masks.
#   C) Only after those feature-side artifacts are locked may official RUL be opened.
#   D) Official RUL is reporting-only: no retraining, recalibration, threshold
#      modification, policy modification, or architecture selection is allowed.
#
# THIS STEP READS NEITHER TEST FEATURES NOR OFFICIAL RUL.

import hashlib
import json


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ax", False)):
    raise RuntimeError(
        "STEP 6.29AX must pass before STEP 6.29AY."
    )

required_629ay = [
    "FD003_PRETEST_EVIDENCE_CLOSURE_629AX",
    "FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX",
    "FD003_DEV_STAGE_CONCLUSION_SHA256_629AX",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
    "FD003_FROZEN_RISK_THRESHOLD_629AU",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW",
]

missing_629ay = [
    name for name in required_629ay
    if name not in globals()
]

if missing_629ay:
    raise RuntimeError(
        "Missing STEP 6.29AY object(s): "
        + ", ".join(missing_629ay)
    )

if not bool(
    globals().get(
        "FD003_PRETEST_EVIDENCE_CLOSED_629AX",
        False,
    )
):
    raise RuntimeError(
        "FD003 pre-TEST evidence is not closed."
    )

if not bool(
    globals().get(
        "FD003_READY_FOR_TEST_ADMISSION_CONTRACT_629AX",
        False,
    )
):
    raise RuntimeError(
        "FD003 is not certified ready for a TEST admission contract."
    )

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AX", True)):
    raise RuntimeError(
        "Unexpected TEST-access state before STEP 6.29AY."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AX", True)):
    raise RuntimeError(
        "Unexpected official-RUL-access state before STEP 6.29AY."
    )

if bool(
    globals().get(
        "FD003_TEST_ADMISSION_CONTRACT_LOCKED_629AY",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AY has already been completed."
    )


# =============================================================================
# 2) Canonical helper
# =============================================================================

def _sha_json_629ay(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify final DEV closure + operating policy
# =============================================================================

if (
    _sha_json_629ay(
        FD003_PRETEST_EVIDENCE_CLOSURE_629AX
    )
    !=
    FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX
):
    raise RuntimeError(
        "STEP 6.29AX pre-TEST closure changed."
    )

if (
    _sha_json_629ay(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    !=
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "STEP 6.29AU operating policy changed before TEST admission."
    )

if not bool(
    FD003_PRETEST_EVIDENCE_CLOSURE_629AX[
        "all_model_training_complete"
    ]
):
    raise RuntimeError(
        "Model training is not certified complete."
    )

if not bool(
    FD003_PRETEST_EVIDENCE_CLOSURE_629AX[
        "risk_architecture_frozen"
    ]
):
    raise RuntimeError(
        "Reliability-risk architecture is not certified frozen."
    )

if not bool(
    FD003_PRETEST_EVIDENCE_CLOSURE_629AX[
        "operating_threshold_frozen"
    ]
):
    raise RuntimeError(
        "Operating threshold is not certified frozen."
    )

if not bool(
    FD003_PRETEST_EVIDENCE_CLOSURE_629AX[
        "operating_policy_frozen"
    ]
):
    raise RuntimeError(
        "Operating policy is not certified frozen."
    )


# =============================================================================
# 4) Freeze one-way held-out TEST admission contract
# =============================================================================

FD003_TEST_ADMISSION_CONTRACT_629AY = {

    "step":
        "6.29AY",

    "dataset":
        "FD003",

    "research_problem":
        "Reliability-Aware Selective Remaining Useful Life Prediction "
        "under Sensor Degradation",

    "pretest_evidence_closure_sha256":
        FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,

    "dev_stage_conclusion_sha256":
        FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,

    "fixed_policy_reporting_contract_sha256":
        FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "test_feature_source_path":
        r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\test_FD003.txt",

    "official_rul_source_path":
        r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\RUL_FD003.txt",

    "test_feature_contents_read_in_this_step":
        False,

    "official_rul_contents_read_in_this_step":
        False,

    "expected_test_engine_count":
        100,

    "expected_test_engine_ids":
        "ascending contiguous integer engine IDs 1..100",

    "raw_test_column_count":
        26,

    "raw_schema":
        [
            "unit_id",
            "cycle",
            "op_setting_1",
            "op_setting_2",
            "op_setting_3",
            "sensor_1",
            "sensor_2",
            "sensor_3",
            "sensor_4",
            "sensor_5",
            "sensor_6",
            "sensor_7",
            "sensor_8",
            "sensor_9",
            "sensor_10",
            "sensor_11",
            "sensor_12",
            "sensor_13",
            "sensor_14",
            "sensor_15",
            "sensor_16",
            "sensor_17",
            "sensor_18",
            "sensor_19",
            "sensor_20",
            "sensor_21",
        ],

    "selected_raw_features":
        [
            "op_setting_1",
            "op_setting_2",
            "sensor_2",
            "sensor_3",
            "sensor_4",
            "sensor_6",
            "sensor_7",
            "sensor_8",
            "sensor_9",
            "sensor_10",
            "sensor_11",
            "sensor_12",
            "sensor_13",
            "sensor_14",
            "sensor_15",
            "sensor_17",
            "sensor_20",
            "sensor_21",
        ],

    "selected_feature_count":
        18,

    "preprocessing_contract_sha256":
        "66d9c90d2cd16ecee4c4d8b7c3585334369ecdc85a8a5709d035d1ea47497853",

    "window_contract_sha256":
        "db0b26953d8eceb3b7cd9541d7dd0bdacfd97e08ddf455fc665e0caccad0bd2e",

    "normalization":
        "use the already-frozen FD003 FIT-only mean/std vectors; no refit, "
        "no TEST-derived normalization statistics",

    "normalization_ddof":
        0,

    "window_length":
        30,

    "window_stride":
        1,

    "test_prediction_population":
        "one terminal 30-cycle window per TEST engine, ending at that engine's "
        "last observed TEST cycle",

    "test_endpoint_order":
        "ascending engine ID 1..100",

    "test_endpoint_count_expected":
        100,

    "short_engine_rule":
        "if any TEST engine has fewer than 30 observed cycles, stop before inference",

    "model_architecture_contract_sha256":
        "613180eb5df3cd9931d26db90c26bebb446ad2f2606b90bef1019f5563eb9644",

    "training_protocol_sha256":
        "15998da0f030e69231b91c656d831c93d352afffcc2f9a52d8ded39ae2e76592",

    "trained_model_state_sha256":
        "c0c39600b67148fa2dce3067a6cc190cababcb69924988f2b9db32e9cd66fa65",

    "deterministic_inference":
        "exactly one deterministic endpoint prediction vector from the frozen model",

    "mc_dropout_passes":
        50,

    "mc_dropout_batch_size":
        512,

    "mc_dropout_seed":
        42,

    "mc_std_ddof":
        0,

    "mc_matrix_retention":
        "not required after TEST uncertainty summary is fingerprinted",

    "calibration_protocol_sha256":
        "e04da22fcbe6b081c9b536c54566c13bbd059567880bbaa1eafbd36b27ef02c2",

    "fit_calibration_evidence_sha256":
        "30c281bf3d9dd4dbca65fff2ecffc050ea6a3806b9649ee58342bcc98c39b40f",

    "g1_contract_sha256":
        "9bb5d1a56a0d2aeda42db844ebfc49bdca260d9a9903ff8e5c3dd1510ff1a06e",

    "clean_selective_risk_contract_sha256":
        "7449a560a1a031de27b32fb5fc81f7402930cd0aff508be7dfe360626332c15d",

    "phase_edge_convention":
        "FIT-only np.quantile(float64, q=[0.2,0.4,0.6,0.8], method='linear')",

    "phase_assignment":
        "np.digitize(..., right=True)",

    "phase_adjustment":
        "MC STD transformed using frozen FIT-only within-phase empirical midrank references",

    "g1_transform":
        "frozen clean-DEV right-ECDF reference",

    "g2_transform":
        "frozen clean-DEV equal-engine-weighted empirical midrank reference",

    "risk_orientation":
        "lower final selective risk = more reliable = accept",

    "known_phase_edge_provenance_limitation":
        "historical exact floating pathway differed by at most 1 ULP for one recovered "
        "reference edge; FD003 linear-quantile convention was prospectively frozen "
        "before FD003 calibration and remains unchanged",

    "policy_form":
        "uncertainty_only_fixed_threshold",

    "frozen_risk_threshold":
        float(
            FD003_FROZEN_RISK_THRESHOLD_629AU
        ),

    "acceptance_comparator":
        "<=",

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "test_threshold_recalibration_allowed":
        False,

    "test_coverage_target_enforcement_allowed":
        False,

    "test_risk_reranking_or_posthoc_tie_break_allowed":
        False,

    "test_realized_coverage_rule":
        "report realized coverage produced by the frozen <= threshold; do not force 0.8",

    "one_way_sequence":
        [
            "1. Open and structurally verify TEST features only.",
            "2. Apply frozen FIT-only feature selection and normalization.",
            "3. Build exactly one terminal 30-cycle endpoint window per TEST engine.",
            "4. Run frozen deterministic inference and canonical 50-pass MC dropout.",
            "5. Apply frozen phase adjustment, G1 transform, and G2 transform.",
            "6. Apply frozen numerical threshold to create TEST accept/abstain masks.",
            "7. Fingerprint and lock TEST endpoint IDs, predictions, MC STD, final risk, "
            "accept mask, abstain mask, and feature-side execution contract.",
            "8. Only after Step 7 is locked may official RUL_FD003.txt be opened.",
            "9. Verify official RUL length/alignment, fingerprint it, and score once.",
            "10. Report held-out TEST metrics without any model/risk/threshold/policy change.",
        ],

    "official_rul_open_gate":
        "official RUL may be opened only after TEST predictions, uncertainty, final risk, "
        "and fixed-threshold decisions are fingerprinted and locked",

    "heldout_test_metrics":
        [
            "endpoint_count",
            "realized_coverage",
            "full_MAE",
            "median_absolute_error",
            "RMSE",
            "accepted_MAE",
            "abstained_MAE",
            "selective_MAE_gain = full_MAE - accepted_MAE",
            "relative_MAE_reduction_pct = 100 * selective_MAE_gain / full_MAE",
            "abstained_minus_accepted_MAE",
        ],

    "empty_subset_rule":
        "if accepted or abstained endpoint subset is empty, corresponding subset metrics "
        "are NaN; threshold/policy is not changed",

    "test_results_may_retrain_model":
        False,

    "test_results_may_modify_preprocessing":
        False,

    "test_results_may_modify_risk_architecture":
        False,

    "test_results_may_modify_threshold":
        False,

    "test_results_may_modify_policy":
        False,

    "test_results_may_modify_corruption_manifest":
        False,

    "test_results_role":
        "one-way held-out evaluation and reporting only",

    "test_admission_contract_created":
        True,

    "test_admission_performed":
        False,

    "test_feature_contents_read":
        False,

    "test_feature_side_artifacts_locked":
        False,

    "official_rul_contents_read":
        False,

    "heldout_test_scoring_performed":
        False,
}


FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY = _sha_json_629ay(
    FD003_TEST_ADMISSION_CONTRACT_629AY
)


# =============================================================================
# 5) Lock pre-open TEST admission state
# =============================================================================

FD003_TEST_ADMISSION_CONTRACT_LOCKED_629AY = True
FD003_TEST_ADMISSION_CONTRACT_CREATED_629AY = True

FD003_TEST_ADMISSION_PERFORMED_629AY = False
FD003_TEST_CONTENTS_READ_629AY = False
FD003_TEST_FEATURE_SIDE_ARTIFACTS_LOCKED_629AY = False
FD003_RUL_CONTENTS_READ_629AY = False
FD003_HELDOUT_TEST_SCORING_PERFORMED_629AY = False

FD003_MODEL_RETRAINED_629AY = False
FD003_PREPROCESSING_MODIFIED_629AY = False
FD003_RISK_ARCHITECTURE_MODIFIED_629AY = False
FD003_THRESHOLD_MODIFIED_629AY = False
FD003_POLICY_MODIFIED_629AY = False

all_checks_passed_629ay = True


# =============================================================================
# 6) Output
# =============================================================================

print("=" * 166)

print(
    "STEP 6.29AY — FD003 ONE-WAY TEST ADMISSION CONTRACT FREEZE — PRE-OPEN"
)

print("=" * 166)

print("TEST admission contract created:", True)
print("TEST admission performed        :", False)
print("TEST feature contents read      :", False)
print("TEST feature artifacts locked   :", False)
print("Official RUL contents read      :", False)
print("Held-out TEST scoring performed :", False)
print("Model retrained                 :", False)
print("Preprocessing modified          :", False)
print("Risk architecture modified      :", False)
print("Threshold modified              :", False)
print("Policy modified                 :", False)

print("\n--- FROZEN ONE-WAY TEST DESIGN ---")

print(
    "Expected TEST engines           :",
    100,
)

print(
    "Endpoint population             :",
    "one terminal 30-cycle window per engine",
)

print(
    "Feature count                   :",
    18,
)

print(
    "MC passes / batch / seed / ddof :",
    (
        50,
        512,
        42,
        0,
    ),
)

print(
    "Frozen risk threshold           :",
    FD003_FROZEN_RISK_THRESHOLD_629AU,
)

print(
    "Comparator                      :",
    "<=",
)

print(
    "Official-RUL opening gate       :",
    "only after TEST prediction/risk/masks are fingerprinted and locked",
)

print(
    "TEST-driven retuning allowed    :",
    False,
)

print(
    "\nTEST-admission contract SHA-256 :",
    FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,
)

print(
    "\nPASS — the FD003 held-out TEST evaluation path is now prospectively "
    "frozen before reading any TEST feature values."
)

print(
    "Official RUL remains sealed. Next: open TEST FEATURES ONLY, perform structural "
    "verification and frozen preprocessing/window reconstruction, and stop before "
    "model inference if any admission check fails."
)

In [ ]:
# STEP 6.29AZ
# FD003 TEST-FEATURE-ONLY ADMISSION + FROZEN PREPROCESSING / TERMINAL-WINDOW LOCK
#
# Opens TEST FEATURES ONLY.
# NO model inference.
# NO MC-dropout.
# NO official RUL access.
# NO held-out scoring.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ay", False)):
    raise RuntimeError("STEP 6.29AY must pass before STEP 6.29AZ.")

required_629az = [
    "FD003_TEST_ADMISSION_CONTRACT_629AY",
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY",
    "FD003_SELECTED_RAW_FEATURES_629E",
    "FD003_FIT_RAW_629C",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "_state_dict_sha256_629k",
]

missing_629az = [
    name for name in required_629az
    if name not in globals()
]

if missing_629az:
    raise RuntimeError(
        "Missing STEP 6.29AZ object(s): "
        + ", ".join(missing_629az)
    )

if not bool(
    globals().get("FD003_TEST_ADMISSION_CONTRACT_LOCKED_629AY", False)
):
    raise RuntimeError("STEP 6.29AY TEST admission contract is not locked.")

if bool(globals().get("FD003_TEST_CONTENTS_READ_629AY", True)):
    raise RuntimeError(
        "TEST contents were unexpectedly marked read before STEP 6.29AZ."
    )

if bool(globals().get("FD003_RUL_CONTENTS_READ_629AY", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly marked read before STEP 6.29AZ."
    )

if bool(
    globals().get("FD003_TEST_FEATURE_RECONSTRUCTION_LOCKED_629AZ", False)
):
    raise RuntimeError("STEP 6.29AZ has already been completed.")


# =============================================================================
# 2) Helpers
# =============================================================================

def _sha_arr_629az(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during hashing.")

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629az(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify frozen admission contract + trained model
# =============================================================================

if (
    _sha_json_629az(FD003_TEST_ADMISSION_CONTRACT_629AY)
    != FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY
):
    raise RuntimeError("STEP 6.29AY TEST admission contract changed.")

admission_629az = FD003_TEST_ADMISSION_CONTRACT_629AY

selected_features_629az = list(
    FD003_SELECTED_RAW_FEATURES_629E
)

if selected_features_629az != list(
    admission_629az["selected_raw_features"]
):
    raise RuntimeError(
        "Frozen selected-feature order differs from TEST admission contract."
    )

if len(selected_features_629az) != 18:
    raise RuntimeError("Frozen selected-feature count is not 18.")

model_sha_before_629az = _state_dict_sha256_629k(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
)

if (
    model_sha_before_629az
    != FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
):
    raise RuntimeError(
        "Frozen FD003 model state changed before TEST admission."
    )


# =============================================================================
# 4) Reconstruct and verify exact frozen FIT-only normalization vectors
# =============================================================================

EXPECTED_FIT_MEANS_SHA256_629AZ = (
    "da6f264ee675bcf34a38bc805877d06f8445d521407fc71ed50f13b13e68c096"
)

EXPECTED_FIT_STDS_SHA256_629AZ = (
    "8d893e3e3b872cb4e59d2d3790c891a32ec5b55516655cf0a4dd75e39a61359a"
)

fit_matrix_629az = FD003_FIT_RAW_629C[
    selected_features_629az
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if fit_matrix_629az.shape != (19434, 18):
    raise RuntimeError(
        "Authoritative FD003 FIT feature matrix shape changed."
    )

fit_means_629az = np.mean(
    fit_matrix_629az,
    axis=0,
    dtype=np.float64,
)

fit_stds_629az = np.std(
    fit_matrix_629az,
    axis=0,
    ddof=0,
    dtype=np.float64,
)

if (
    _sha_arr_629az(fit_means_629az)
    != EXPECTED_FIT_MEANS_SHA256_629AZ
):
    raise RuntimeError(
        "FIT means do not match the frozen normalization fingerprint."
    )

if (
    _sha_arr_629az(fit_stds_629az)
    != EXPECTED_FIT_STDS_SHA256_629AZ
):
    raise RuntimeError(
        "FIT stds do not match the frozen normalization fingerprint."
    )

if not np.isfinite(fit_means_629az).all():
    raise RuntimeError("FIT means contain non-finite values.")

if not np.isfinite(fit_stds_629az).all():
    raise RuntimeError("FIT stds contain non-finite values.")

if not bool((fit_stds_629az > 0.0).all()):
    raise RuntimeError("FIT stds contain non-positive values.")


# =============================================================================
# 5) OPEN TEST FEATURES ONLY
# =============================================================================

test_path_629az = admission_629az["test_feature_source_path"]
raw_schema_629az = list(admission_629az["raw_schema"])

if len(raw_schema_629az) != 26:
    raise RuntimeError("Frozen raw TEST schema does not contain 26 columns.")

test_raw_629az = pd.read_csv(
    test_path_629az,
    sep=r"\s+",
    header=None,
    names=raw_schema_629az,
    engine="python",
)

FD003_TEST_CONTENTS_READ_629AZ = True
FD003_RUL_CONTENTS_READ_629AZ = False


# =============================================================================
# 6) Strict TEST structural verification
# =============================================================================

if test_raw_629az.shape[1] != 26:
    raise RuntimeError("TEST raw column count is not 26.")

if list(test_raw_629az.columns) != raw_schema_629az:
    raise RuntimeError(
        "TEST raw column order differs from the frozen schema."
    )

raw_values_629az = test_raw_629az.to_numpy(
    dtype=np.float64,
    copy=True,
)

if not np.isfinite(raw_values_629az).all():
    raise RuntimeError("TEST raw frame contains non-finite values.")

unit_float_629az = test_raw_629az["unit_id"].to_numpy(
    dtype=np.float64
)

cycle_float_629az = test_raw_629az["cycle"].to_numpy(
    dtype=np.float64
)

if not np.array_equal(
    unit_float_629az,
    unit_float_629az.astype(np.int64),
):
    raise RuntimeError("TEST unit_id contains non-integer values.")

if not np.array_equal(
    cycle_float_629az,
    cycle_float_629az.astype(np.int64),
):
    raise RuntimeError("TEST cycle contains non-integer values.")

test_raw_629az["unit_id"] = unit_float_629az.astype(np.int64)
test_raw_629az["cycle"] = cycle_float_629az.astype(np.int64)

test_engine_ids_629az = (
    test_raw_629az["unit_id"]
    .drop_duplicates()
    .to_numpy(dtype=np.int64)
)

expected_engine_ids_629az = np.arange(
    1,
    101,
    dtype=np.int64,
)

if not np.array_equal(
    test_engine_ids_629az,
    expected_engine_ids_629az,
):
    raise RuntimeError(
        "TEST engine IDs are not exactly ordered 1..100."
    )

if int(test_raw_629az["unit_id"].nunique()) != 100:
    raise RuntimeError("TEST engine count is not exactly 100.")

if bool(
    test_raw_629az.duplicated(
        subset=["unit_id", "cycle"]
    ).any()
):
    raise RuntimeError(
        "Duplicate TEST (unit_id, cycle) rows detected."
    )

engine_lengths_629az = {}

for engine_id_629az in expected_engine_ids_629az:

    engine_df_629az = test_raw_629az.loc[
        test_raw_629az["unit_id"] == engine_id_629az
    ]

    cycles_629az = engine_df_629az["cycle"].to_numpy(
        dtype=np.int64
    )

    if cycles_629az.size == 0:
        raise RuntimeError(
            f"TEST engine {engine_id_629az} has no rows."
        )

    if cycles_629az[0] != 1:
        raise RuntimeError(
            f"TEST engine {engine_id_629az} does not start at cycle 1."
        )

    if not np.array_equal(
        cycles_629az,
        np.arange(
            1,
            cycles_629az[-1] + 1,
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            f"TEST engine {engine_id_629az} cycles are not contiguous/ordered."
        )

    if cycles_629az.size < 30:
        raise RuntimeError(
            f"TEST engine {engine_id_629az} has fewer than 30 cycles."
        )

    engine_lengths_629az[int(engine_id_629az)] = int(
        cycles_629az.size
    )


# =============================================================================
# 7) Frozen FIT-only preprocessing
# =============================================================================

test_selected_raw_629az = test_raw_629az[
    selected_features_629az
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if test_selected_raw_629az.shape != (
    len(test_raw_629az),
    18,
):
    raise RuntimeError(
        "TEST selected-feature matrix shape changed."
    )

test_normalized_629az = (
    test_selected_raw_629az
    - fit_means_629az.reshape(1, -1)
) / fit_stds_629az.reshape(1, -1)

if not np.isfinite(test_normalized_629az).all():
    raise RuntimeError(
        "Normalized TEST feature matrix contains non-finite values."
    )


# =============================================================================
# 8) One terminal 30-cycle window per TEST engine
# =============================================================================

terminal_windows_629az = []
endpoint_engine_ids_629az = []
endpoint_cycles_629az = []

all_units_629az = test_raw_629az["unit_id"].to_numpy(
    dtype=np.int64
)

for engine_id_629az in expected_engine_ids_629az:

    engine_mask_629az = (
        all_units_629az == engine_id_629az
    )

    engine_norm_629az = test_normalized_629az[
        engine_mask_629az
    ]

    engine_cycles_629az = test_raw_629az.loc[
        engine_mask_629az,
        "cycle",
    ].to_numpy(
        dtype=np.int64
    )

    if engine_norm_629az.shape[0] < 30:
        raise RuntimeError(
            f"TEST engine {engine_id_629az} unexpectedly has <30 rows."
        )

    terminal_windows_629az.append(
        engine_norm_629az[-30:]
    )

    endpoint_engine_ids_629az.append(
        int(engine_id_629az)
    )

    endpoint_cycles_629az.append(
        int(engine_cycles_629az[-1])
    )


FD003_TEST_ENDPOINT_WINDOWS_629AZ = np.asarray(
    terminal_windows_629az,
    dtype=np.float32,
)

FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ = np.asarray(
    endpoint_engine_ids_629az,
    dtype=np.int64,
)

FD003_TEST_ENDPOINT_CYCLES_629AZ = np.asarray(
    endpoint_cycles_629az,
    dtype=np.int64,
)


if FD003_TEST_ENDPOINT_WINDOWS_629AZ.shape != (100, 30, 18):
    raise RuntimeError(
        "TEST endpoint-window tensor shape is not (100, 30, 18)."
    )

if not np.array_equal(
    FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ,
    expected_engine_ids_629az,
):
    raise RuntimeError(
        "TEST endpoint engine order is not exactly 1..100."
    )

for i_629az, engine_id_629az in enumerate(
    expected_engine_ids_629az
):

    expected_cycle_629az = engine_lengths_629az[
        int(engine_id_629az)
    ]

    if (
        int(FD003_TEST_ENDPOINT_CYCLES_629AZ[i_629az])
        != expected_cycle_629az
    ):
        raise RuntimeError(
            f"TEST endpoint cycle mismatch for engine {engine_id_629az}."
        )


# =============================================================================
# 9) Fingerprints
# =============================================================================

FD003_TEST_RAW_NUMERIC_SHA256_629AZ = _sha_arr_629az(
    test_raw_629az[
        raw_schema_629az
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )
)

FD003_TEST_SELECTED_RAW_SHA256_629AZ = _sha_arr_629az(
    test_selected_raw_629az
)

FD003_TEST_NORMALIZED_SHA256_629AZ = _sha_arr_629az(
    test_normalized_629az
)

FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ = _sha_arr_629az(
    FD003_TEST_ENDPOINT_WINDOWS_629AZ
)

FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ = _sha_arr_629az(
    FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ
)

FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ = _sha_arr_629az(
    FD003_TEST_ENDPOINT_CYCLES_629AZ
)


# =============================================================================
# 10) Confirm model untouched
# =============================================================================

model_sha_after_629az = _state_dict_sha256_629k(
    FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH
)

if model_sha_after_629az != model_sha_before_629az:
    raise RuntimeError(
        "FD003 model state changed during TEST reconstruction."
    )


# =============================================================================
# 11) Freeze TEST reconstruction contract
# =============================================================================

length_values_629az = np.asarray(
    list(engine_lengths_629az.values()),
    dtype=np.int64,
)

FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_629AZ = {

    "step": "6.29AZ",
    "dataset": "FD003",

    "test_admission_contract_sha256":
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,

    "test_feature_source_path":
        test_path_629az,

    "official_rul_contents_read":
        False,

    "test_raw_shape":
        [
            int(test_raw_629az.shape[0]),
            int(test_raw_629az.shape[1]),
        ],

    "test_engine_count":
        100,

    "engine_cycle_length_min":
        int(length_values_629az.min()),

    "engine_cycle_length_median":
        float(np.median(length_values_629az)),

    "engine_cycle_length_mean":
        float(np.mean(length_values_629az)),

    "engine_cycle_length_max":
        int(length_values_629az.max()),

    "selected_raw_features":
        selected_features_629az,

    "fit_means_sha256":
        EXPECTED_FIT_MEANS_SHA256_629AZ,

    "fit_stds_sha256":
        EXPECTED_FIT_STDS_SHA256_629AZ,

    "normalization_ddof":
        0,

    "window_length":
        30,

    "endpoint_count":
        100,

    "endpoint_rule":
        "one terminal 30-cycle window per engine at last observed TEST cycle",

    "raw_numeric_sha256":
        FD003_TEST_RAW_NUMERIC_SHA256_629AZ,

    "selected_raw_sha256":
        FD003_TEST_SELECTED_RAW_SHA256_629AZ,

    "normalized_matrix_sha256":
        FD003_TEST_NORMALIZED_SHA256_629AZ,

    "endpoint_windows_shape":
        [100, 30, 18],

    "endpoint_windows_dtype":
        "float32",

    "endpoint_windows_sha256":
        FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ,

    "endpoint_engine_ids_sha256":
        FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ,

    "endpoint_cycles_sha256":
        FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ,

    "model_state_before_sha256":
        model_sha_before_629az,

    "model_state_after_sha256":
        model_sha_after_629az,

    "model_state_unchanged":
        True,

    "model_inference_performed":
        False,

    "mc_dropout_execution_performed":
        False,

    "test_feature_contents_read":
        True,

    "test_feature_side_artifacts_locked":
        False,

    "official_rul_contents_read":
        False,

    "heldout_test_scoring_performed":
        False,
}


FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_SHA256_629AZ = (
    _sha_json_629az(
        FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_629AZ
    )
)


# =============================================================================
# 12) Read-only locks + state
# =============================================================================

FD003_TEST_ENDPOINT_WINDOWS_629AZ.setflags(write=False)
FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ.setflags(write=False)
FD003_TEST_ENDPOINT_CYCLES_629AZ.setflags(write=False)

FD003_TEST_FEATURE_RECONSTRUCTION_LOCKED_629AZ = True
FD003_TEST_FEATURES_ADMITTED_629AZ = True

FD003_TEST_FEATURE_ARTIFACTS_COMPLETE_629AZ = False
FD003_TEST_MODEL_INFERENCE_PERFORMED_629AZ = False
FD003_TEST_MC_EXECUTION_PERFORMED_629AZ = False

FD003_RUL_CONTENTS_READ_629AZ = False
FD003_HELDOUT_TEST_SCORING_PERFORMED_629AZ = False

all_checks_passed_629az = True


# =============================================================================
# 13) Output
# =============================================================================

print("=" * 168)

print(
    "STEP 6.29AZ — FD003 TEST-FEATURE-ONLY ADMISSION + "
    "FROZEN PREPROCESSING / TERMINAL-WINDOW LOCK"
)

print("=" * 168)

print("TEST feature contents read      :", True)
print("Official RUL contents read      :", False)
print("Model inference performed       :", False)
print("MC-dropout execution performed  :", False)
print("Held-out TEST scoring performed :", False)

print("\n--- TEST STRUCTURE ---")

print(
    "Raw TEST shape                  :",
    test_raw_629az.shape,
)

print(
    "TEST engines                    :",
    len(FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ),
)

print(
    "Engine IDs exact 1..100         :",
    np.array_equal(
        FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ,
        expected_engine_ids_629az,
    ),
)

print(
    "Engine length min/median/mean/max:",
    (
        int(length_values_629az.min()),
        float(np.median(length_values_629az)),
        float(np.mean(length_values_629az)),
        int(length_values_629az.max()),
    ),
)

print("\n--- FROZEN PREPROCESSING ---")

print("Selected feature count          :", 18)
print("FIT means fingerprint exact     :", True)
print("FIT stds fingerprint exact      :", True)

print(
    "Normalized TEST shape           :",
    test_normalized_629az.shape,
)

print("\n--- TERMINAL ENDPOINT WINDOWS ---")

print(
    "Endpoint window tensor shape    :",
    FD003_TEST_ENDPOINT_WINDOWS_629AZ.shape,
)

print(
    "Endpoint engine-ID shape        :",
    FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ.shape,
)

print(
    "Endpoint-cycle shape            :",
    FD003_TEST_ENDPOINT_CYCLES_629AZ.shape,
)

print(
    "Endpoint windows SHA-256        :",
    FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ,
)

print(
    "Endpoint engine IDs SHA-256     :",
    FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ,
)

print(
    "Endpoint cycles SHA-256         :",
    FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ,
)

print(
    "Model state unchanged           :",
    model_sha_after_629az == model_sha_before_629az,
)

print(
    "\nTEST reconstruction contract SHA-256:",
    FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_SHA256_629AZ,
)

print(
    "\nPASS — held-out FD003 TEST features were admitted under the "
    "prospectively frozen contract and reconstructed into exactly 100 "
    "terminal model-input windows using only frozen FIT-derived preprocessing."
)

print(
    "Official RUL remains sealed. Next: run the frozen model + canonical "
    "MC-dropout + phase→G1→G2 reliability chain on these 100 endpoints, "
    "apply the already-frozen threshold, fingerprint all TEST feature-side "
    "artifacts, and only then permit official RUL access."
)

In [ ]:
# STEP 6.29BA
# FD003 HELD-OUT TEST FEATURE-SIDE INFERENCE + RELIABILITY / DECISION LOCK
#
# Uses ONLY the already-admitted TEST feature windows from STEP 6.29AZ.
#
# Executes:
#   - frozen deterministic model inference
#   - canonical 50-pass MC dropout
#   - frozen phase adjustment -> G1 -> G2 reliability chain
#   - already-frozen fixed threshold
#
# Locks/fingerprints all feature-side TEST artifacts BEFORE official RUL access.
#
# NO official RUL access.
# NO TEST error scoring.
# NO retraining, recalibration, threshold/policy modification.

import hashlib
import json
import numpy as np
import torch


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629az", False)):
    raise RuntimeError(
        "STEP 6.29AZ must pass before STEP 6.29BA."
    )

required_629ba = [
    "FD003_TEST_ADMISSION_CONTRACT_629AY",
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY",
    "FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_629AZ",
    "FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_SHA256_629AZ",
    "FD003_TEST_ENDPOINT_WINDOWS_629AZ",
    "FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ",
    "FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ",
    "FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ",
    "FD003_TEST_ENDPOINT_CYCLES_629AZ",
    "FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH",
    "FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH",
    "_state_dict_sha256_629k",
    "mc_dropout_predict_fd003_629m",
    "apply_fd003_phase_adjustment_629s",
    "apply_fd003_g1_right_ecdf_629s",
    "apply_fd003_g2_weighted_midrank_629v",
    "FD003_FROZEN_RISK_THRESHOLD_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
]

missing_629ba = [
    name for name in required_629ba
    if name not in globals()
]

if missing_629ba:
    raise RuntimeError(
        "Missing STEP 6.29BA object(s): "
        + ", ".join(missing_629ba)
    )

if not bool(
    globals().get(
        "FD003_TEST_FEATURE_RECONSTRUCTION_LOCKED_629AZ",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AZ TEST reconstruction is not locked."
    )

if not bool(
    globals().get(
        "FD003_TEST_FEATURES_ADMITTED_629AZ",
        False,
    )
):
    raise RuntimeError(
        "TEST features were not admitted by STEP 6.29AZ."
    )

if not bool(
    globals().get(
        "FD003_TEST_CONTENTS_READ_629AZ",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29AZ did not mark TEST features as read."
    )

if bool(
    globals().get(
        "FD003_RUL_CONTENTS_READ_629AZ",
        True,
    )
):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.29BA."
    )

if bool(
    globals().get(
        "FD003_TEST_FEATURE_SIDE_ARTIFACTS_LOCKED_629BA",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29BA has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629ba(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29BA hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629ba(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify frozen contracts and TEST endpoint identities
# =============================================================================

if (
    _sha_json_629ba(
        FD003_TEST_ADMISSION_CONTRACT_629AY
    )
    != FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY
):
    raise RuntimeError(
        "STEP 6.29AY TEST admission contract changed."
    )

if (
    _sha_json_629ba(
        FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_629AZ
    )
    != FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_SHA256_629AZ
):
    raise RuntimeError(
        "STEP 6.29AZ TEST reconstruction contract changed."
    )

if (
    _sha_json_629ba(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    != FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "Frozen STEP 6.29AU operating policy changed."
    )

if (
    _sha_arr_629ba(
        FD003_TEST_ENDPOINT_WINDOWS_629AZ
    )
    != FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ
):
    raise RuntimeError(
        "Frozen TEST endpoint-window tensor changed."
    )

if (
    _sha_arr_629ba(
        FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ
    )
    != FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ
):
    raise RuntimeError(
        "Frozen TEST endpoint engine IDs changed."
    )

if (
    _sha_arr_629ba(
        FD003_TEST_ENDPOINT_CYCLES_629AZ
    )
    != FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ
):
    raise RuntimeError(
        "Frozen TEST endpoint cycles changed."
    )


# =============================================================================
# 4) Re-certify trained model state and mode
# =============================================================================

model_629ba = FD003_AUTHORITATIVE_BRIDGE_MODEL_629AH

model_sha_before_629ba = _state_dict_sha256_629k(
    model_629ba
)

if (
    model_sha_before_629ba
    != FD003_AUTHORITATIVE_BRIDGE_MODEL_STATE_SHA256_629AH
):
    raise RuntimeError(
        "Frozen FD003 model state changed before TEST inference."
    )

if bool(model_629ba.training):
    raise RuntimeError(
        "Frozen FD003 model is unexpectedly in training mode "
        "before TEST inference."
    )

device_629ba = next(
    model_629ba.parameters()
).device


# =============================================================================
# 5) Prepare writable TEST tensor copy
# =============================================================================

X_test_629ba = np.array(
    FD003_TEST_ENDPOINT_WINDOWS_629AZ,
    dtype=np.float32,
    copy=True,
)

if X_test_629ba.shape != (100, 30, 18):
    raise RuntimeError(
        "TEST endpoint input shape is not (100, 30, 18)."
    )

if not np.isfinite(X_test_629ba).all():
    raise RuntimeError(
        "TEST endpoint input contains non-finite values."
    )


# =============================================================================
# 6) Frozen deterministic endpoint inference
# =============================================================================

model_629ba.eval()

det_batches_629ba = []

with torch.no_grad():

    for start_629ba in range(
        0,
        X_test_629ba.shape[0],
        512,
    ):

        stop_629ba = min(
            start_629ba + 512,
            X_test_629ba.shape[0],
        )

        batch_629ba = torch.from_numpy(
            X_test_629ba[
                start_629ba:stop_629ba
            ]
        ).to(
            device_629ba
        )

        pred_629ba = model_629ba(
            batch_629ba
        )

        det_batches_629ba.append(
            pred_629ba.detach().cpu().numpy()
        )

FD003_TEST_DETERMINISTIC_PREDICTION_629BA = np.concatenate(
    det_batches_629ba,
    axis=0,
).reshape(-1)

if FD003_TEST_DETERMINISTIC_PREDICTION_629BA.shape != (100,):
    raise RuntimeError(
        "TEST deterministic prediction shape is not (100,)."
    )

if not np.isfinite(
    FD003_TEST_DETERMINISTIC_PREDICTION_629BA
).all():
    raise RuntimeError(
        "TEST deterministic predictions contain non-finite values."
    )


# =============================================================================
# 7) Canonical frozen 50-pass MC-dropout execution
# =============================================================================

(
    mc_matrix_629ba,
    FD003_TEST_MC_MEAN_629BA,
    FD003_TEST_MC_STD_629BA,
    mc_runtime_seconds_629ba,
) = mc_dropout_predict_fd003_629m(
    X_test_629ba,
    model_629ba,
    device_629ba,
    passes=50,
    batch_size=512,
    seed=42,
)

mc_matrix_629ba = np.asarray(
    mc_matrix_629ba
)

FD003_TEST_MC_MEAN_629BA = np.asarray(
    FD003_TEST_MC_MEAN_629BA
).reshape(-1)

FD003_TEST_MC_STD_629BA = np.asarray(
    FD003_TEST_MC_STD_629BA
).reshape(-1)

if mc_matrix_629ba.shape != (50, 100):
    raise RuntimeError(
        "TEST MC matrix shape is not (50, 100)."
    )

if FD003_TEST_MC_MEAN_629BA.shape != (100,):
    raise RuntimeError(
        "TEST MC-mean shape is not (100,)."
    )

if FD003_TEST_MC_STD_629BA.shape != (100,):
    raise RuntimeError(
        "TEST MC-STD shape is not (100,)."
    )

if not np.isfinite(mc_matrix_629ba).all():
    raise RuntimeError(
        "TEST MC matrix contains non-finite values."
    )

if not np.isfinite(
    FD003_TEST_MC_MEAN_629BA
).all():
    raise RuntimeError(
        "TEST MC mean contains non-finite values."
    )

if not np.isfinite(
    FD003_TEST_MC_STD_629BA
).all():
    raise RuntimeError(
        "TEST MC STD contains non-finite values."
    )

if not bool(
    (
        FD003_TEST_MC_STD_629BA
        >=
        0.0
    ).all()
):
    raise RuntimeError(
        "TEST MC STD contains negative values."
    )


# =============================================================================
# 8) Frozen label-free reliability chain: phase -> G1 -> G2
# =============================================================================

FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_629BA = np.asarray(
    apply_fd003_phase_adjustment_629s(
        FD003_TEST_DETERMINISTIC_PREDICTION_629BA,
        FD003_TEST_MC_STD_629BA,
    )
).reshape(-1)

FD003_TEST_G1_RISK_629BA = np.asarray(
    apply_fd003_g1_right_ecdf_629s(
        FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_629BA
    )
).reshape(-1)

FD003_TEST_FINAL_SELECTIVE_RISK_629BA = np.asarray(
    apply_fd003_g2_weighted_midrank_629v(
        FD003_TEST_G1_RISK_629BA
    )
).reshape(-1)

for name_629ba, vec_629ba in (
    (
        "phase-adjusted uncertainty",
        FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_629BA,
    ),
    (
        "G1 risk",
        FD003_TEST_G1_RISK_629BA,
    ),
    (
        "final selective risk",
        FD003_TEST_FINAL_SELECTIVE_RISK_629BA,
    ),
):

    if vec_629ba.shape != (100,):
        raise RuntimeError(
            f"TEST {name_629ba} shape is not (100,)."
        )

    if not np.isfinite(vec_629ba).all():
        raise RuntimeError(
            f"TEST {name_629ba} contains non-finite values."
        )

if not bool(
    (
        (FD003_TEST_G1_RISK_629BA >= 0.0)
        &
        (FD003_TEST_G1_RISK_629BA <= 1.0)
    ).all()
):
    raise RuntimeError(
        "TEST G1 risk is outside [0,1]."
    )

if not bool(
    (
        (FD003_TEST_FINAL_SELECTIVE_RISK_629BA >= 0.0)
        &
        (FD003_TEST_FINAL_SELECTIVE_RISK_629BA <= 1.0)
    ).all()
):
    raise RuntimeError(
        "TEST final selective risk is outside [0,1]."
    )


# =============================================================================
# 9) Apply already-frozen numerical threshold
# =============================================================================

threshold_629ba = float(
    FD003_FROZEN_RISK_THRESHOLD_629AU
)

if (
    threshold_629ba
    !=
    float(
        FD003_OPERATING_POLICY_CONTRACT_629AU[
            "frozen_risk_threshold"
        ]
    )
):
    raise RuntimeError(
        "Frozen TEST threshold identity changed."
    )

if (
    str(
        FD003_OPERATING_POLICY_CONTRACT_629AU[
            "acceptance_comparator"
        ]
    )
    !=
    "<="
):
    raise RuntimeError(
        "Frozen TEST comparator changed."
    )

FD003_TEST_ACCEPT_MASK_629BA = (
    FD003_TEST_FINAL_SELECTIVE_RISK_629BA
    <=
    threshold_629ba
)

FD003_TEST_ABSTAIN_MASK_629BA = (
    ~FD003_TEST_ACCEPT_MASK_629BA
)

accepted_count_629ba = int(
    np.count_nonzero(
        FD003_TEST_ACCEPT_MASK_629BA
    )
)

abstained_count_629ba = int(
    np.count_nonzero(
        FD003_TEST_ABSTAIN_MASK_629BA
    )
)

realized_coverage_629ba = float(
    accepted_count_629ba / 100.0
)

below_count_629ba = int(
    np.count_nonzero(
        FD003_TEST_FINAL_SELECTIVE_RISK_629BA
        <
        threshold_629ba
    )
)

equal_count_629ba = int(
    np.count_nonzero(
        FD003_TEST_FINAL_SELECTIVE_RISK_629BA
        ==
        threshold_629ba
    )
)

above_count_629ba = int(
    np.count_nonzero(
        FD003_TEST_FINAL_SELECTIVE_RISK_629BA
        >
        threshold_629ba
    )
)

if (
    below_count_629ba
    +
    equal_count_629ba
    +
    above_count_629ba
    !=
    100
):
    raise RuntimeError(
        "TEST risk threshold partition does not cover all 100 endpoints."
    )

if accepted_count_629ba != (
    below_count_629ba
    +
    equal_count_629ba
):
    raise RuntimeError(
        "TEST accept count is inconsistent with frozen <= comparator."
    )


# =============================================================================
# 10) Feature-side fingerprints
# =============================================================================

FD003_TEST_DETERMINISTIC_PREDICTION_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_DETERMINISTIC_PREDICTION_629BA
)

FD003_TEST_MC_MATRIX_SHA256_629BA = _sha_arr_629ba(
    mc_matrix_629ba
)

FD003_TEST_MC_MEAN_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_MC_MEAN_629BA
)

FD003_TEST_MC_STD_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_MC_STD_629BA
)

FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_629BA
)

FD003_TEST_G1_RISK_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_G1_RISK_629BA
)

FD003_TEST_FINAL_SELECTIVE_RISK_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_FINAL_SELECTIVE_RISK_629BA
)

FD003_TEST_ACCEPT_MASK_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_ACCEPT_MASK_629BA
)

FD003_TEST_ABSTAIN_MASK_SHA256_629BA = _sha_arr_629ba(
    FD003_TEST_ABSTAIN_MASK_629BA
)


# =============================================================================
# 11) Confirm model state/mode restored
# =============================================================================

model_sha_after_629ba = _state_dict_sha256_629k(
    model_629ba
)

if model_sha_after_629ba != model_sha_before_629ba:
    raise RuntimeError(
        "FD003 model state changed during held-out TEST inference."
    )

if bool(model_629ba.training):
    raise RuntimeError(
        "FD003 model mode was not restored to eval after TEST MC dropout."
    )


# =============================================================================
# 12) Lock feature-side TEST execution contract
# =============================================================================

FD003_TEST_FEATURE_SIDE_CONTRACT_629BA = {

    "step":
        "6.29BA",

    "dataset":
        "FD003",

    "test_admission_contract_sha256":
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,

    "test_reconstruction_contract_sha256":
        FD003_TEST_FEATURE_RECONSTRUCTION_CONTRACT_SHA256_629AZ,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "endpoint_count":
        100,

    "endpoint_engine_ids_sha256":
        FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ,

    "endpoint_cycles_sha256":
        FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ,

    "endpoint_windows_sha256":
        FD003_TEST_ENDPOINT_WINDOWS_SHA256_629AZ,

    "model_state_before_sha256":
        model_sha_before_629ba,

    "model_state_after_sha256":
        model_sha_after_629ba,

    "model_state_unchanged":
        True,

    "model_mode_restored_eval":
        True,

    "deterministic_prediction_sha256":
        FD003_TEST_DETERMINISTIC_PREDICTION_SHA256_629BA,

    "mc_passes":
        50,

    "mc_batch_size":
        512,

    "mc_seed":
        42,

    "mc_std_ddof":
        0,

    "mc_matrix_shape":
        [50, 100],

    "mc_matrix_sha256":
        FD003_TEST_MC_MATRIX_SHA256_629BA,

    "mc_mean_sha256":
        FD003_TEST_MC_MEAN_SHA256_629BA,

    "mc_std_sha256":
        FD003_TEST_MC_STD_SHA256_629BA,

    "mc_runtime_seconds":
        float(mc_runtime_seconds_629ba),

    "phase_adjusted_uncertainty_sha256":
        FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_SHA256_629BA,

    "g1_risk_sha256":
        FD003_TEST_G1_RISK_SHA256_629BA,

    "final_selective_risk_sha256":
        FD003_TEST_FINAL_SELECTIVE_RISK_SHA256_629BA,

    "frozen_risk_threshold":
        threshold_629ba,

    "acceptance_comparator":
        "<=",

    "accepted_count":
        accepted_count_629ba,

    "abstained_count":
        abstained_count_629ba,

    "realized_test_coverage":
        realized_coverage_629ba,

    "below_threshold_count":
        below_count_629ba,

    "equal_threshold_count":
        equal_count_629ba,

    "above_threshold_count":
        above_count_629ba,

    "accept_mask_sha256":
        FD003_TEST_ACCEPT_MASK_SHA256_629BA,

    "abstain_mask_sha256":
        FD003_TEST_ABSTAIN_MASK_SHA256_629BA,

    "test_feature_contents_read":
        True,

    "official_rul_contents_read":
        False,

    "heldout_test_scoring_performed":
        False,

    "model_retrained":
        False,

    "preprocessing_modified":
        False,

    "risk_architecture_modified":
        False,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "official_rul_open_gate_satisfied":
        True,
}


FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA = _sha_json_629ba(
    FD003_TEST_FEATURE_SIDE_CONTRACT_629BA
)


# =============================================================================
# 13) Read-only artifact locks
# =============================================================================

for arr_629ba in (
    FD003_TEST_DETERMINISTIC_PREDICTION_629BA,
    FD003_TEST_MC_MEAN_629BA,
    FD003_TEST_MC_STD_629BA,
    FD003_TEST_PHASE_ADJUSTED_UNCERTAINTY_629BA,
    FD003_TEST_G1_RISK_629BA,
    FD003_TEST_FINAL_SELECTIVE_RISK_629BA,
    FD003_TEST_ACCEPT_MASK_629BA,
    FD003_TEST_ABSTAIN_MASK_629BA,
):

    arr_629ba.setflags(
        write=False
    )

FD003_TEST_FEATURE_SIDE_ARTIFACTS_LOCKED_629BA = True
FD003_TEST_FEATURE_SIDE_EXECUTION_COMPLETE_629BA = True
FD003_OFFICIAL_RUL_OPEN_GATE_SATISFIED_629BA = True

FD003_TEST_MODEL_INFERENCE_PERFORMED_629BA = True
FD003_TEST_MC_EXECUTION_PERFORMED_629BA = True

FD003_TEST_CONTENTS_READ_629BA = True
FD003_RUL_CONTENTS_READ_629BA = False
FD003_HELDOUT_TEST_SCORING_PERFORMED_629BA = False

FD003_MODEL_RETRAINED_629BA = False
FD003_PREPROCESSING_MODIFIED_629BA = False
FD003_RISK_ARCHITECTURE_MODIFIED_629BA = False
FD003_THRESHOLD_MODIFIED_629BA = False
FD003_POLICY_MODIFIED_629BA = False

all_checks_passed_629ba = True


# =============================================================================
# 14) Discard full MC matrix after fingerprinting
# =============================================================================

del mc_matrix_629ba


# =============================================================================
# 15) Output
# =============================================================================

print("=" * 170)

print(
    "STEP 6.29BA — FD003 HELD-OUT TEST FEATURE-SIDE "
    "INFERENCE + RELIABILITY / DECISION LOCK"
)

print("=" * 170)

print("TEST features used               :", True)
print("Deterministic inference performed:", True)
print("MC-dropout execution performed   :", True)
print("Official RUL contents read       :", False)
print("Held-out TEST scoring performed  :", False)
print("Model retrained                  :", False)
print("Preprocessing modified           :", False)
print("Risk architecture modified       :", False)
print("Threshold modified               :", False)
print("Policy modified                  :", False)

print(
    "\n--- TEST FEATURE-SIDE ARTIFACTS ---"
)

print(
    "Endpoint count                  :",
    100,
)

print(
    "Prediction shape                :",
    FD003_TEST_DETERMINISTIC_PREDICTION_629BA.shape,
)

print(
    "MC mean / std shapes            :",
    (
        FD003_TEST_MC_MEAN_629BA.shape,
        FD003_TEST_MC_STD_629BA.shape,
    ),
)

print(
    "Final risk shape                :",
    FD003_TEST_FINAL_SELECTIVE_RISK_629BA.shape,
)

print(
    "Prediction SHA-256              :",
    FD003_TEST_DETERMINISTIC_PREDICTION_SHA256_629BA,
)

print(
    "MC matrix SHA-256               :",
    FD003_TEST_MC_MATRIX_SHA256_629BA,
)

print(
    "MC STD SHA-256                  :",
    FD003_TEST_MC_STD_SHA256_629BA,
)

print(
    "Final risk SHA-256              :",
    FD003_TEST_FINAL_SELECTIVE_RISK_SHA256_629BA,
)

print(
    "\n--- FROZEN TEST DECISIONS ---"
)

print(
    "Frozen threshold                :",
    threshold_629ba,
)

print(
    "Below / equal / above threshold :",
    (
        below_count_629ba,
        equal_count_629ba,
        above_count_629ba,
    ),
)

print(
    "Accepted / abstained            :",
    (
        accepted_count_629ba,
        abstained_count_629ba,
    ),
)

print(
    "Realized TEST coverage          :",
    realized_coverage_629ba,
)

print(
    "Accept-mask SHA-256             :",
    FD003_TEST_ACCEPT_MASK_SHA256_629BA,
)

print(
    "Abstain-mask SHA-256            :",
    FD003_TEST_ABSTAIN_MASK_SHA256_629BA,
)

print(
    "\n--- SAFETY / ONE-WAY GATE ---"
)

print(
    "Model state unchanged           :",
    model_sha_after_629ba
    ==
    model_sha_before_629ba,
)

print(
    "Model restored to eval          :",
    not bool(model_629ba.training),
)

print(
    "Feature-side artifacts locked   :",
    True,
)

print(
    "Official-RUL open gate satisfied:",
    True,
)

print(
    "TEST feature-side contract SHA-256:",
    FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,
)

print(
    "\nPASS — all held-out FD003 TEST feature-side predictions, uncertainty, "
    "final reliability risk, and fixed-threshold accept/abstain decisions "
    "are now fingerprinted and locked before official RUL access."
)

print(
    "Official RUL is still unopened. Next: open RUL_FD003.txt exactly once, "
    "verify its 100-engine alignment, fingerprint it, and score the already-locked "
    "TEST predictions/decisions without changing anything."
)

In [ ]:
# STEP 6.29BB
# FD003 OFFICIAL-RUL ONE-WAY OPEN + HELD-OUT TEST SCORING LOCK

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629ba", False)):
    raise RuntimeError(
        "STEP 6.29BA must pass before STEP 6.29BB."
    )

required_629bb = [
    "FD003_TEST_ADMISSION_CONTRACT_629AY",
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA",
    "FD003_TEST_DETERMINISTIC_PREDICTION_629BA",
    "FD003_TEST_DETERMINISTIC_PREDICTION_SHA256_629BA",
    "FD003_TEST_FINAL_SELECTIVE_RISK_629BA",
    "FD003_TEST_FINAL_SELECTIVE_RISK_SHA256_629BA",
    "FD003_TEST_ACCEPT_MASK_629BA",
    "FD003_TEST_ACCEPT_MASK_SHA256_629BA",
    "FD003_TEST_ABSTAIN_MASK_629BA",
    "FD003_TEST_ABSTAIN_MASK_SHA256_629BA",
    "FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ",
    "FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ",
    "FD003_TEST_ENDPOINT_CYCLES_629AZ",
    "FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
]

missing_629bb = [
    name for name in required_629bb
    if name not in globals()
]

if missing_629bb:
    raise RuntimeError(
        "Missing STEP 6.29BB object(s): "
        + ", ".join(missing_629bb)
    )

if not bool(
    globals().get(
        "FD003_TEST_FEATURE_SIDE_ARTIFACTS_LOCKED_629BA",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29BA TEST feature-side artifacts are not locked."
    )

if not bool(
    globals().get(
        "FD003_OFFICIAL_RUL_OPEN_GATE_SATISFIED_629BA",
        False,
    )
):
    raise RuntimeError(
        "Official-RUL open gate is not satisfied."
    )

if bool(
    globals().get(
        "FD003_RUL_CONTENTS_READ_629BA",
        True,
    )
):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.29BB."
    )

if bool(
    globals().get(
        "FD003_HELDOUT_TEST_SCORING_LOCKED_629BB",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29BB has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_arr_629bb(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.29BB hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_629bb(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _json_scalar_629bb(value):

    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (str, bool, int, float)):
        return value

    raise TypeError(
        "Unsupported canonical scalar type: "
        + type(value).__name__
    )


def _df_sha256_629bb(df):

    return _sha_json_629bb(
        {
            "columns": [
                str(c)
                for c in df.columns
            ],
            "records": [
                [
                    _json_scalar_629bb(x)
                    for x in row
                ]
                for row in df.itertuples(
                    index=False,
                    name=None,
                )
            ],
        }
    )


# =============================================================================
# 3) Re-certify all pre-label TEST artifacts
# =============================================================================

if (
    _sha_json_629bb(
        FD003_TEST_ADMISSION_CONTRACT_629AY
    )
    !=
    FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY
):
    raise RuntimeError(
        "STEP 6.29AY TEST admission contract changed."
    )

if (
    _sha_json_629bb(
        FD003_TEST_FEATURE_SIDE_CONTRACT_629BA
    )
    !=
    FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA
):
    raise RuntimeError(
        "STEP 6.29BA TEST feature-side contract changed."
    )

if (
    _sha_json_629bb(
        FD003_OPERATING_POLICY_CONTRACT_629AU
    )
    !=
    FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU
):
    raise RuntimeError(
        "Frozen operating-policy contract changed."
    )

for value_629bb, expected_sha_629bb, label_629bb in (
    (
        FD003_TEST_DETERMINISTIC_PREDICTION_629BA,
        FD003_TEST_DETERMINISTIC_PREDICTION_SHA256_629BA,
        "TEST deterministic prediction",
    ),
    (
        FD003_TEST_FINAL_SELECTIVE_RISK_629BA,
        FD003_TEST_FINAL_SELECTIVE_RISK_SHA256_629BA,
        "TEST final selective risk",
    ),
    (
        FD003_TEST_ACCEPT_MASK_629BA,
        FD003_TEST_ACCEPT_MASK_SHA256_629BA,
        "TEST accept mask",
    ),
    (
        FD003_TEST_ABSTAIN_MASK_629BA,
        FD003_TEST_ABSTAIN_MASK_SHA256_629BA,
        "TEST abstain mask",
    ),
    (
        FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ,
        FD003_TEST_ENDPOINT_ENGINE_IDS_SHA256_629AZ,
        "TEST endpoint engine IDs",
    ),
    (
        FD003_TEST_ENDPOINT_CYCLES_629AZ,
        FD003_TEST_ENDPOINT_CYCLES_SHA256_629AZ,
        "TEST endpoint cycles",
    ),
):

    if (
        _sha_arr_629bb(value_629bb)
        !=
        expected_sha_629bb
    ):
        raise RuntimeError(
            label_629bb
            + " fingerprint changed before official-RUL opening."
        )

feature_contract_629bb = (
    FD003_TEST_FEATURE_SIDE_CONTRACT_629BA
)

if not bool(
    feature_contract_629bb[
        "official_rul_open_gate_satisfied"
    ]
):
    raise RuntimeError(
        "Locked feature-side contract does not permit official-RUL opening."
    )

if bool(
    feature_contract_629bb[
        "official_rul_contents_read"
    ]
):
    raise RuntimeError(
        "Locked feature-side contract says official RUL was already read."
    )

if bool(
    feature_contract_629bb[
        "heldout_test_scoring_performed"
    ]
):
    raise RuntimeError(
        "Locked feature-side contract says held-out scoring already occurred."
    )


# =============================================================================
# 4) OPEN OFFICIAL RUL EXACTLY ONCE
# =============================================================================

rul_path_629bb = (
    FD003_TEST_ADMISSION_CONTRACT_629AY[
        "official_rul_source_path"
    ]
)

rul_raw_629bb = pd.read_csv(
    rul_path_629bb,
    sep=r"\s+",
    header=None,
    engine="python",
)

FD003_RUL_CONTENTS_READ_629BB = True

if rul_raw_629bb.shape != (100, 1):
    raise RuntimeError(
        "Official RUL_FD003.txt shape is not exactly (100, 1)."
    )

rul_vector_629bb = (
    rul_raw_629bb.iloc[:, 0]
    .to_numpy(
        dtype=np.float64,
        copy=True,
    )
)

if rul_vector_629bb.shape != (100,):
    raise RuntimeError(
        "Official RUL vector shape is not (100,)."
    )

if not np.isfinite(
    rul_vector_629bb
).all():
    raise RuntimeError(
        "Official RUL vector contains non-finite values."
    )

if not bool(
    (
        rul_vector_629bb
        >=
        0.0
    ).all()
):
    raise RuntimeError(
        "Official RUL vector contains negative values."
    )

if not np.array_equal(
    rul_vector_629bb,
    rul_vector_629bb.astype(
        np.int64
    ),
):
    raise RuntimeError(
        "Official RUL vector contains non-integer values."
    )


# =============================================================================
# 5) Verify engine alignment
# =============================================================================

engine_ids_629bb = np.asarray(
    FD003_TEST_ENDPOINT_ENGINE_IDS_629AZ,
    dtype=np.int64,
).reshape(-1)

endpoint_cycles_629bb = np.asarray(
    FD003_TEST_ENDPOINT_CYCLES_629AZ,
    dtype=np.int64,
).reshape(-1)

if not np.array_equal(
    engine_ids_629bb,
    np.arange(
        1,
        101,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "TEST endpoint engine IDs are no longer exactly 1..100."
    )

# CMAPSS RUL file order corresponds TEST engine order 1..100.
FD003_OFFICIAL_RUL_VECTOR_629BB = np.array(
    rul_vector_629bb,
    dtype=np.float64,
    copy=True,
)

FD003_OFFICIAL_RUL_VECTOR_SHA256_629BB = _sha_arr_629bb(
    FD003_OFFICIAL_RUL_VECTOR_629BB
)


# =============================================================================
# 6) Score only the already-locked predictions and decisions
# =============================================================================

pred_629bb = np.asarray(
    FD003_TEST_DETERMINISTIC_PREDICTION_629BA,
    dtype=np.float64,
).reshape(-1)

risk_629bb = np.asarray(
    FD003_TEST_FINAL_SELECTIVE_RISK_629BA,
    dtype=np.float64,
).reshape(-1)

accept_629bb = np.asarray(
    FD003_TEST_ACCEPT_MASK_629BA,
    dtype=bool,
).reshape(-1)

abstain_629bb = np.asarray(
    FD003_TEST_ABSTAIN_MASK_629BA,
    dtype=bool,
).reshape(-1)

if not (
    pred_629bb.shape
    ==
    risk_629bb.shape
    ==
    accept_629bb.shape
    ==
    abstain_629bb.shape
    ==
    (100,)
):
    raise RuntimeError(
        "Locked TEST feature-side vectors do not all have shape (100,)."
    )

if not np.array_equal(
    abstain_629bb,
    ~accept_629bb,
):
    raise RuntimeError(
        "Locked TEST abstain mask is not the complement of accept mask."
    )


FD003_TEST_ABS_ERROR_629BB = np.abs(
    pred_629bb
    -
    FD003_OFFICIAL_RUL_VECTOR_629BB
)

FD003_TEST_ABS_ERROR_SHA256_629BB = _sha_arr_629bb(
    FD003_TEST_ABS_ERROR_629BB
)


full_mae_629bb = float(
    np.mean(
        FD003_TEST_ABS_ERROR_629BB
    )
)

median_abs_error_629bb = float(
    np.median(
        FD003_TEST_ABS_ERROR_629BB
    )
)

rmse_629bb = float(
    np.sqrt(
        np.mean(
            (
                pred_629bb
                -
                FD003_OFFICIAL_RUL_VECTOR_629BB
            ) ** 2
        )
    )
)

accepted_count_629bb = int(
    np.count_nonzero(
        accept_629bb
    )
)

abstained_count_629bb = int(
    np.count_nonzero(
        abstain_629bb
    )
)

realized_coverage_629bb = float(
    accepted_count_629bb
    /
    100.0
)

accepted_mae_629bb = (
    float(
        np.mean(
            FD003_TEST_ABS_ERROR_629BB[
                accept_629bb
            ]
        )
    )
    if accepted_count_629bb > 0
    else float("nan")
)

abstained_mae_629bb = (
    float(
        np.mean(
            FD003_TEST_ABS_ERROR_629BB[
                abstain_629bb
            ]
        )
    )
    if abstained_count_629bb > 0
    else float("nan")
)

selective_gain_629bb = (
    float(
        full_mae_629bb
        -
        accepted_mae_629bb
    )
    if accepted_count_629bb > 0
    else float("nan")
)

relative_reduction_pct_629bb = (
    float(
        100.0
        *
        selective_gain_629bb
        /
        full_mae_629bb
    )
    if (
        accepted_count_629bb > 0
        and
        full_mae_629bb != 0.0
    )
    else float("nan")
)

abstained_minus_accepted_mae_629bb = (
    float(
        abstained_mae_629bb
        -
        accepted_mae_629bb
    )
    if (
        accepted_count_629bb > 0
        and
        abstained_count_629bb > 0
    )
    else float("nan")
)


# =============================================================================
# 7) Lock aligned endpoint-level result table
# =============================================================================

FD003_TEST_ENDPOINT_RESULTS_629BB = pd.DataFrame(
    {
        "engine_id":
            engine_ids_629bb,

        "endpoint_cycle":
            endpoint_cycles_629bb,

        "prediction":
            pred_629bb,

        "final_selective_risk":
            risk_629bb,

        "accepted":
            accept_629bb,

        "official_rul":
            FD003_OFFICIAL_RUL_VECTOR_629BB,

        "absolute_error":
            FD003_TEST_ABS_ERROR_629BB,
    }
)

FD003_TEST_ENDPOINT_RESULTS_SHA256_629BB = _df_sha256_629bb(
    FD003_TEST_ENDPOINT_RESULTS_629BB
)


# =============================================================================
# 8) Freeze held-out TEST score contract
# =============================================================================

FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB = {

    "step":
        "6.29BB",

    "dataset":
        "FD003",

    "test_admission_contract_sha256":
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,

    "test_feature_side_contract_sha256":
        FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "official_rul_source_path":
        rul_path_629bb,

    "official_rul_shape":
        [100],

    "official_rul_alignment":
        "RUL row i corresponds to frozen TEST engine ID i+1; "
        "TEST endpoint engine order verified exactly 1..100",

    "official_rul_sha256":
        FD003_OFFICIAL_RUL_VECTOR_SHA256_629BB,

    "endpoint_results_sha256":
        FD003_TEST_ENDPOINT_RESULTS_SHA256_629BB,

    "absolute_error_sha256":
        FD003_TEST_ABS_ERROR_SHA256_629BB,

    "endpoint_count":
        100,

    "accepted_count":
        accepted_count_629bb,

    "abstained_count":
        abstained_count_629bb,

    "realized_coverage":
        realized_coverage_629bb,

    "full_mae":
        full_mae_629bb,

    "median_absolute_error":
        median_abs_error_629bb,

    "rmse":
        rmse_629bb,

    "accepted_mae":
        accepted_mae_629bb,

    "abstained_mae":
        abstained_mae_629bb,

    "selective_mae_gain":
        selective_gain_629bb,

    "relative_mae_reduction_pct":
        relative_reduction_pct_629bb,

    "abstained_minus_accepted_mae":
        abstained_minus_accepted_mae_629bb,

    "official_rul_contents_read":
        True,

    "heldout_test_scoring_performed":
        True,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "model_retrained":
        False,

    "preprocessing_modified":
        False,

    "risk_architecture_modified":
        False,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "test_results_used_for_selection_or_tuning":
        False,
}


FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB = (
    _sha_json_629bb(
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB
    )
)


# =============================================================================
# 9) Read-only locks + state
# =============================================================================

FD003_OFFICIAL_RUL_VECTOR_629BB.setflags(
    write=False
)

FD003_TEST_ABS_ERROR_629BB.setflags(
    write=False
)

FD003_HELDOUT_TEST_SCORING_LOCKED_629BB = True
FD003_HELDOUT_TEST_SCORING_PERFORMED_629BB = True
FD003_RUL_CONTENTS_READ_629BB = True

FD003_NEW_MODEL_INFERENCE_PERFORMED_629BB = False
FD003_NEW_MC_EXECUTION_PERFORMED_629BB = False
FD003_MODEL_RETRAINED_629BB = False
FD003_PREPROCESSING_MODIFIED_629BB = False
FD003_RISK_ARCHITECTURE_MODIFIED_629BB = False
FD003_THRESHOLD_MODIFIED_629BB = False
FD003_POLICY_MODIFIED_629BB = False
FD003_TEST_RESULTS_USED_FOR_TUNING_629BB = False

all_checks_passed_629bb = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 170)

print(
    "STEP 6.29BB — FD003 OFFICIAL-RUL ONE-WAY OPEN "
    "+ HELD-OUT TEST SCORING LOCK"
)

print("=" * 170)

print("Official RUL contents read       :", True)
print("Held-out TEST scoring performed  :", True)
print("New model inference performed    :", False)
print("New MC-dropout execution         :", False)
print("Model retrained                  :", False)
print("Preprocessing modified           :", False)
print("Risk architecture modified       :", False)
print("Threshold modified               :", False)
print("Policy modified                  :", False)
print("TEST results used for tuning     :", False)

print("\n--- OFFICIAL RUL AUTHORITY ---")

print(
    "Official RUL shape               :",
    FD003_OFFICIAL_RUL_VECTOR_629BB.shape,
)

print(
    "RUL min / median / mean / max    :",
    (
        float(
            np.min(
                FD003_OFFICIAL_RUL_VECTOR_629BB
            )
        ),
        float(
            np.median(
                FD003_OFFICIAL_RUL_VECTOR_629BB
            )
        ),
        float(
            np.mean(
                FD003_OFFICIAL_RUL_VECTOR_629BB
            )
        ),
        float(
            np.max(
                FD003_OFFICIAL_RUL_VECTOR_629BB
            )
        ),
    ),
)

print(
    "Engine alignment exact 1..100    :",
    np.array_equal(
        engine_ids_629bb,
        np.arange(
            1,
            101,
            dtype=np.int64,
        ),
    ),
)

print(
    "Official RUL SHA-256             :",
    FD003_OFFICIAL_RUL_VECTOR_SHA256_629BB,
)

print("\n--- FINAL HELD-OUT TEST METRICS ---")

print(
    "Endpoint count                   :",
    100,
)

print(
    "Accepted / abstained             :",
    (
        accepted_count_629bb,
        abstained_count_629bb,
    ),
)

print(
    "Realized coverage                :",
    realized_coverage_629bb,
)

print(
    "Full TEST MAE                    :",
    full_mae_629bb,
)

print(
    "Median absolute error            :",
    median_abs_error_629bb,
)

print(
    "TEST RMSE                        :",
    rmse_629bb,
)

print(
    "Accepted TEST MAE                :",
    accepted_mae_629bb,
)

print(
    "Abstained TEST MAE               :",
    abstained_mae_629bb,
)

print(
    "Selective MAE gain               :",
    selective_gain_629bb,
)

print(
    "Relative MAE reduction (%)       :",
    relative_reduction_pct_629bb,
)

print(
    "Abstained - accepted MAE         :",
    abstained_minus_accepted_mae_629bb,
)

print(
    "\nAbsolute-error SHA-256           :",
    FD003_TEST_ABS_ERROR_SHA256_629BB,
)

print(
    "Endpoint-results SHA-256         :",
    FD003_TEST_ENDPOINT_RESULTS_SHA256_629BB,
)

print(
    "Held-out score contract SHA-256  :",
    FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB,
)

print(
    "\nPASS — official FD003 RUL was opened only after TEST feature-side "
    "predictions, uncertainty, risk, and accept/abstain decisions were locked, "
    "and the held-out TEST evaluation has now been scored exactly once."
)

print(
    "No model, preprocessing, risk construction, threshold, or policy was changed. "
    "Next: interpret and lock the final FD003 held-out conclusion and close FD003 "
    "before moving to FD004."
)

In [ ]:
# STEP 6.29BC
# FD003 FINAL HELD-OUT SCIENTIFIC CONCLUSION + DATASET CLOSURE LOCK
#
# Interprets and fingerprints the already-locked STEP 6.29BB held-out result.
#
# NO new scoring.
# NO new inference / MC / corruption.
# NO retraining, recalibration, threshold modification, or policy modification.
# TEST results are reporting-only.
#
# After this step, FD003 is CLOSED and may not be retuned.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629bb", False)):
    raise RuntimeError(
        "STEP 6.29BB must pass before STEP 6.29BC."
    )

required_629bc = [
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA",
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",
    "FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX",
    "FD003_DEV_STAGE_CONCLUSION_629AX",
    "FD003_DEV_STAGE_CONCLUSION_SHA256_629AX",
    "FD003_OFFICIAL_RUL_VECTOR_SHA256_629BB",
    "FD003_TEST_ENDPOINT_RESULTS_SHA256_629BB",
    "FD003_TEST_ABS_ERROR_SHA256_629BB",
]

missing_629bc = [
    name
    for name in required_629bc
    if name not in globals()
]

if missing_629bc:
    raise RuntimeError(
        "Missing STEP 6.29BC object(s): "
        + ", ".join(missing_629bc)
    )

if not bool(
    globals().get(
        "FD003_HELDOUT_TEST_SCORING_LOCKED_629BB",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29BB held-out TEST scoring is not locked."
    )

if not bool(
    globals().get(
        "FD003_RUL_CONTENTS_READ_629BB",
        False,
    )
):
    raise RuntimeError(
        "Official RUL was not opened by STEP 6.29BB."
    )

if bool(
    globals().get(
        "FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.29BC has already been completed."
    )


# =============================================================================
# 2) Canonical helper
# =============================================================================

def _sha_json_629bc(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify held-out score + DEV conclusion
# =============================================================================

if (
    _sha_json_629bc(
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB
    )
    !=
    FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB
):
    raise RuntimeError(
        "STEP 6.29BB held-out score contract changed."
    )

if (
    _sha_json_629bc(
        FD003_DEV_STAGE_CONCLUSION_629AX
    )
    !=
    FD003_DEV_STAGE_CONCLUSION_SHA256_629AX
):
    raise RuntimeError(
        "STEP 6.29AX DEV-stage conclusion changed."
    )

test_629bc = FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB
dev_629bc = FD003_DEV_STAGE_CONCLUSION_629AX


# =============================================================================
# 4) Extract already-locked TEST evidence only
# =============================================================================

endpoint_count_629bc = int(
    test_629bc["endpoint_count"]
)

accepted_count_629bc = int(
    test_629bc["accepted_count"]
)

abstained_count_629bc = int(
    test_629bc["abstained_count"]
)

coverage_629bc = float(
    test_629bc["realized_coverage"]
)

full_mae_629bc = float(
    test_629bc["full_mae"]
)

median_abs_629bc = float(
    test_629bc["median_absolute_error"]
)

rmse_629bc = float(
    test_629bc["rmse"]
)

accepted_mae_629bc = float(
    test_629bc["accepted_mae"]
)

abstained_mae_629bc = float(
    test_629bc["abstained_mae"]
)

selective_gain_629bc = float(
    test_629bc["selective_mae_gain"]
)

relative_reduction_629bc = float(
    test_629bc["relative_mae_reduction_pct"]
)

abstained_minus_accepted_629bc = float(
    test_629bc["abstained_minus_accepted_mae"]
)


# =============================================================================
# 5) Extract already-locked DEV/corruption evidence for comparison only
# =============================================================================

dev_clean_629bc = dev_629bc[
    "clean_dev"
]

dev_corr_629bc = dev_629bc[
    "controlled_corruption_fixed_policy"
]

dev_clean_gain_629bc = float(
    dev_clean_629bc[
        "selective_mae_gain"
    ]
)

dev_clean_relative_reduction_629bc = float(
    dev_clean_629bc[
        "relative_mae_reduction_pct"
    ]
)

corr_positive_rate_629bc = float(
    dev_corr_629bc[
        "positive_gain_rate"
    ]
)

corr_negative_count_629bc = int(
    dev_corr_629bc[
        "negative_gain_count"
    ]
)


# =============================================================================
# 6) Freeze final scientific interpretation
# =============================================================================

if selective_gain_629bc < 0.0:
    selective_test_interpretation_629bc = (
        "The frozen reliability-aware selective policy did not improve held-out "
        "FD003 TEST MAE: accepted-endpoint MAE was slightly higher than full TEST "
        "MAE, while abstained-endpoint MAE was slightly lower than accepted MAE."
    )
elif selective_gain_629bc > 0.0:
    selective_test_interpretation_629bc = (
        "The frozen reliability-aware selective policy improved held-out FD003 "
        "TEST MAE on the accepted subset."
    )
else:
    selective_test_interpretation_629bc = (
        "The frozen reliability-aware selective policy produced no held-out "
        "FD003 TEST MAE change on the accepted subset."
    )


FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC = {

    "step":
        "6.29BC",

    "dataset":
        "FD003",

    "research_problem":
        "Reliability-Aware Selective Remaining Useful Life Prediction "
        "under Sensor Degradation",

    "final_status":
        "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED",

    "heldout_test":
        {
            "endpoint_count":
                endpoint_count_629bc,

            "accepted_count":
                accepted_count_629bc,

            "abstained_count":
                abstained_count_629bc,

            "realized_coverage":
                coverage_629bc,

            "full_mae":
                full_mae_629bc,

            "median_absolute_error":
                median_abs_629bc,

            "rmse":
                rmse_629bc,

            "accepted_mae":
                accepted_mae_629bc,

            "abstained_mae":
                abstained_mae_629bc,

            "selective_mae_gain":
                selective_gain_629bc,

            "relative_mae_reduction_pct":
                relative_reduction_629bc,

            "abstained_minus_accepted_mae":
                abstained_minus_accepted_629bc,
        },

    "dev_context":
        {
            "clean_dev_selective_mae_gain":
                dev_clean_gain_629bc,

            "clean_dev_relative_mae_reduction_pct":
                dev_clean_relative_reduction_629bc,

            "controlled_corruption_positive_gain_rate":
                corr_positive_rate_629bc,

            "controlled_corruption_negative_gain_count":
                corr_negative_count_629bc,
        },

    "heldout_selective_interpretation":
        selective_test_interpretation_629bc,

    "generalization_interpretation":
        (
            "FD003 shows a DEV-to-held-out generalization gap in the reliability "
            "ranking/selective-abstention mechanism. The frozen policy was useful "
            "on clean DEV and in 333 of 336 controlled DEV corruption scenarios, "
            "but that selective MAE advantage did not transfer to the 100 held-out "
            "TEST endpoints under the prospectively frozen threshold."
        ),

    "base_prediction_interpretation":
        (
            "The held-out deterministic RUL predictor itself remains evaluable "
            "independently of the selective mechanism; the final full TEST MAE and "
            "RMSE are reported as observed without post-TEST modification."
        ),

    "claim_boundary":
        (
            "FD003 does not support a claim that the frozen reliability score "
            "universally separates lower-error from higher-error held-out endpoints. "
            "This negative held-out selective result is retained as evidence and "
            "must not be used to retune FD003."
        ),

    "research_alignment":
        (
            "The result remains aligned with the research problem because the study "
            "evaluates both predictive accuracy and the reliability-aware selective "
            "mechanism under a leakage-safe one-way protocol, including unfavorable "
            "held-out outcomes rather than optimizing them away."
        ),

    "post_test_retraining_allowed":
        False,

    "post_test_recalibration_allowed":
        False,

    "post_test_risk_architecture_change_allowed":
        False,

    "post_test_threshold_change_allowed":
        False,

    "post_test_policy_change_allowed":
        False,

    "test_results_role":
        "reporting and cross-dataset scientific synthesis only",

    "official_rul_contents_read":
        True,

    "heldout_test_scoring_complete":
        True,
}


FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC = _sha_json_629bc(
    FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC
)


# =============================================================================
# 7) Freeze final FD003 dataset closure contract
# =============================================================================

FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC = {

    "step":
        "6.29BC",

    "dataset":
        "FD003",

    "status":
        "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED",

    "pretest_evidence_closure_sha256":
        FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,

    "test_admission_contract_sha256":
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,

    "test_feature_side_contract_sha256":
        FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,

    "heldout_test_score_contract_sha256":
        FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB,

    "operating_policy_contract_sha256":
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

    "official_rul_sha256":
        FD003_OFFICIAL_RUL_VECTOR_SHA256_629BB,

    "endpoint_results_sha256":
        FD003_TEST_ENDPOINT_RESULTS_SHA256_629BB,

    "absolute_error_sha256":
        FD003_TEST_ABS_ERROR_SHA256_629BB,

    "final_scientific_conclusion_sha256":
        FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,

    "all_training_complete":
        True,

    "all_calibration_complete":
        True,

    "controlled_corruption_execution_complete":
        True,

    "controlled_corruption_reporting_complete":
        True,

    "operating_policy_frozen_before_test_labels":
        True,

    "test_feature_side_artifacts_locked_before_rul":
        True,

    "official_rul_opened_after_feature_side_lock":
        True,

    "heldout_test_scored_once":
        True,

    "test_results_used_for_tuning":
        False,

    "model_retraining_after_test":
        False,

    "preprocessing_change_after_test":
        False,

    "risk_architecture_change_after_test":
        False,

    "threshold_change_after_test":
        False,

    "policy_change_after_test":
        False,

    "fd003_closed":
        True,

    "next_dataset":
        "FD004",
}


FD003_FINAL_DATASET_CLOSURE_SHA256_629BC = _sha_json_629bc(
    FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC
)


# =============================================================================
# 8) Final immutable state flags
# =============================================================================

FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC = True
FD003_COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED_629BC = True
FD003_CLOSED_629BC = True

FD003_MODEL_RETRAINED_AFTER_TEST_629BC = False
FD003_PREPROCESSING_MODIFIED_AFTER_TEST_629BC = False
FD003_RISK_ARCHITECTURE_MODIFIED_AFTER_TEST_629BC = False
FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC = False
FD003_POLICY_MODIFIED_AFTER_TEST_629BC = False
FD003_TEST_RESULTS_USED_FOR_TUNING_629BC = False

all_checks_passed_629bc = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 172)

print(
    "STEP 6.29BC — FD003 FINAL HELD-OUT SCIENTIFIC CONCLUSION "
    "+ DATASET CLOSURE LOCK"
)

print("=" * 172)

print("Final FD003 status             : COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED")
print("Held-out scoring complete      :", True)
print("TEST results used for tuning   :", False)
print("Model retrained after TEST     :", False)
print("Preprocessing modified         :", False)
print("Risk architecture modified     :", False)
print("Threshold modified             :", False)
print("Policy modified                :", False)

print("\n--- FINAL FD003 HELD-OUT EVIDENCE ---")

print(
    "Endpoints                      :",
    endpoint_count_629bc,
)

print(
    "Accepted / abstained           :",
    (
        accepted_count_629bc,
        abstained_count_629bc,
    ),
)

print(
    "Realized coverage              :",
    coverage_629bc,
)

print(
    "Full TEST MAE                  :",
    full_mae_629bc,
)

print(
    "Median absolute error          :",
    median_abs_629bc,
)

print(
    "TEST RMSE                      :",
    rmse_629bc,
)

print(
    "Accepted TEST MAE              :",
    accepted_mae_629bc,
)

print(
    "Abstained TEST MAE             :",
    abstained_mae_629bc,
)

print(
    "Selective MAE gain             :",
    selective_gain_629bc,
)

print(
    "Relative MAE reduction (%)     :",
    relative_reduction_629bc,
)

print(
    "Abstained - accepted MAE       :",
    abstained_minus_accepted_629bc,
)

print("\n--- DEV -> HELD-OUT RELIABILITY GENERALIZATION ---")

print(
    "Clean DEV selective gain       :",
    dev_clean_gain_629bc,
)

print(
    "Corruption positive-gain rate  :",
    corr_positive_rate_629bc,
)

print(
    "Held-out TEST selective gain   :",
    selective_gain_629bc,
)

print(
    "Held-out selective improvement :",
    selective_gain_629bc > 0.0,
)

print(
    "\nScientific interpretation:"
)

print(
    selective_test_interpretation_629bc
)

print(
    "\nFinal conclusion SHA-256       :",
    FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,
)

print(
    "Final closure SHA-256          :",
    FD003_FINAL_DATASET_CLOSURE_SHA256_629BC,
)

print(
    "\nPASS — FD003 is now permanently closed for model/risk/threshold/policy "
    "selection. The held-out negative selective result is retained exactly as "
    "observed and is available only for reporting and later cross-dataset synthesis."
)

print(
    "Next: begin FD004 with a fresh dataset-entry/no-leakage source-path lock. "
    "Do not inherit FD003 numerical thresholds, calibration references, or TEST results."
)

In [ ]:
# STEP RESTART-0
# FD003 POST-CLOSURE RESTART-SAFE CHECKPOINT
#
# Purpose:
#   Preserve the exact frozen FD003 closure state needed to resume at FD004
#   after a full laptop / Python-kernel restart.
#
# Writes:
#   C:\RUL_Reliability_Research\checkpoints\
#       fd003_post_629bc_restart_checkpoint_v1.json
#       fd003_post_629bc_restart_checkpoint_v1.json.sha256
#
# DOES NOT:
#   - retrain anything
#   - perform inference / MC-dropout
#   - reopen TEST or official RUL
#   - modify threshold / policy / preprocessing / risk architecture

import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_629bc", False)):
    raise RuntimeError(
        "STEP 6.29BC must pass before creating the restart checkpoint."
    )

required_restart0 = [
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",
    "FD003_FINAL_DATASET_CLOSURE_SHA256_629BC",

    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC",

    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB",

    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA",

    "FD003_TEST_ADMISSION_CONTRACT_629AY",
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY",

    "FD003_PRETEST_EVIDENCE_CLOSURE_629AX",
    "FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX",

    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU",

    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW",

    "FD003_DEV_STAGE_CONCLUSION_629AX",
    "FD003_DEV_STAGE_CONCLUSION_SHA256_629AX",
]

missing_restart0 = [
    name
    for name in required_restart0
    if name not in globals()
]

if missing_restart0:
    raise RuntimeError(
        "Missing required frozen object(s): "
        + ", ".join(missing_restart0)
    )

if not bool(
    globals().get(
        "FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC",
        False,
    )
):
    raise RuntimeError(
        "FD003 final closure is not locked."
    )

if not bool(
    globals().get(
        "FD003_CLOSED_629BC",
        False,
    )
):
    raise RuntimeError(
        "FD003 is not marked permanently closed."
    )


# =============================================================================
# 2) Canonical hashing helpers
# =============================================================================

def _canonical_json_bytes_restart0(value):

    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def _sha_json_restart0(value):

    return hashlib.sha256(
        _canonical_json_bytes_restart0(value)
    ).hexdigest()


# =============================================================================
# 3) Re-certify every frozen object BEFORE writing anything
# =============================================================================

frozen_pairs_restart0 = [

    (
        "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",
        FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,
        FD003_FINAL_DATASET_CLOSURE_SHA256_629BC,
    ),

    (
        "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
        FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,
        FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,
    ),

    (
        "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,
        FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB,
    ),

    (
        "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
        FD003_TEST_FEATURE_SIDE_CONTRACT_629BA,
        FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,
    ),

    (
        "FD003_TEST_ADMISSION_CONTRACT_629AY",
        FD003_TEST_ADMISSION_CONTRACT_629AY,
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,
    ),

    (
        "FD003_PRETEST_EVIDENCE_CLOSURE_629AX",
        FD003_PRETEST_EVIDENCE_CLOSURE_629AX,
        FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,
    ),

    (
        "FD003_OPERATING_POLICY_CONTRACT_629AU",
        FD003_OPERATING_POLICY_CONTRACT_629AU,
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,
    ),

    (
        "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
        FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW,
        FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,
    ),

    (
        "FD003_DEV_STAGE_CONCLUSION_629AX",
        FD003_DEV_STAGE_CONCLUSION_629AX,
        FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,
    ),
]


for (
    object_name_restart0,
    object_value_restart0,
    expected_sha_restart0,
) in frozen_pairs_restart0:

    observed_sha_restart0 = _sha_json_restart0(
        object_value_restart0
    )

    if observed_sha_restart0 != expected_sha_restart0:

        raise RuntimeError(
            object_name_restart0
            + " failed fingerprint verification before checkpoint creation."
        )


# =============================================================================
# 4) Build compact restart payload
# =============================================================================

checkpoint_payload_restart0 = {

    "checkpoint_schema":
        "FD003_POST_629BC_RESTART_CHECKPOINT_V1",

    "project":
        "Reliability-Aware Selective Remaining Useful Life Prediction "
        "under Sensor Degradation",

    "resume_point":
        "FD004_STEP_6.30A_PRE_ENTRY",

    "fd003_status":
        "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED",

    "fd003_closed":
        True,

    "next_dataset":
        "FD004",

    "scientific_rules_after_restart":
        {
            "do_not_retrain_fd003":
                True,

            "do_not_retune_fd003":
                True,

            "do_not_change_fd003_preprocessing":
                True,

            "do_not_change_fd003_risk_architecture":
                True,

            "do_not_change_fd003_threshold":
                True,

            "do_not_change_fd003_policy":
                True,

            "fd003_test_results_reporting_only":
                True,

            "fd004_inherit_methodology_only":
                True,

            "fd004_inherit_fd003_numeric_state":
                False,
        },

    "frozen_objects":
        {
            "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC":
                FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,

            "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC":
                FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,

            "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB":
                FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,

            "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA":
                FD003_TEST_FEATURE_SIDE_CONTRACT_629BA,

            "FD003_TEST_ADMISSION_CONTRACT_629AY":
                FD003_TEST_ADMISSION_CONTRACT_629AY,

            "FD003_PRETEST_EVIDENCE_CLOSURE_629AX":
                FD003_PRETEST_EVIDENCE_CLOSURE_629AX,

            "FD003_OPERATING_POLICY_CONTRACT_629AU":
                FD003_OPERATING_POLICY_CONTRACT_629AU,

            "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW":
                FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW,

            "FD003_DEV_STAGE_CONCLUSION_629AX":
                FD003_DEV_STAGE_CONCLUSION_629AX,
        },

    "expected_hashes":
        {
            "FD003_FINAL_DATASET_CLOSURE_SHA256_629BC":
                FD003_FINAL_DATASET_CLOSURE_SHA256_629BC,

            "FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC":
                FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,

            "FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB":
                FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB,

            "FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA":
                FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,

            "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY":
                FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,

            "FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX":
                FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,

            "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU":
                FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,

            "FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW":
                FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,

            "FD003_DEV_STAGE_CONCLUSION_SHA256_629AX":
                FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,
        },

    "restore_flags":
        {
            "all_checks_passed_629bc":
                True,

            "FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC":
                True,

            "FD003_COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED_629BC":
                True,

            "FD003_CLOSED_629BC":
                True,

            "FD003_MODEL_RETRAINED_AFTER_TEST_629BC":
                False,

            "FD003_PREPROCESSING_MODIFIED_AFTER_TEST_629BC":
                False,

            "FD003_RISK_ARCHITECTURE_MODIFIED_AFTER_TEST_629BC":
                False,

            "FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC":
                False,

            "FD003_POLICY_MODIFIED_AFTER_TEST_629BC":
                False,

            "FD003_TEST_RESULTS_USED_FOR_TUNING_629BC":
                False,
        },
}


checkpoint_payload_sha256_restart0 = _sha_json_restart0(
    checkpoint_payload_restart0
)


checkpoint_wrapper_restart0 = {

    "payload":
        checkpoint_payload_restart0,

    "payload_sha256":
        checkpoint_payload_sha256_restart0,
}


# =============================================================================
# 5) Atomic disk write
# =============================================================================

checkpoint_dir_restart0 = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
)

checkpoint_dir_restart0.mkdir(
    parents=True,
    exist_ok=True,
)


checkpoint_path_restart0 = (
    checkpoint_dir_restart0
    /
    "fd003_post_629bc_restart_checkpoint_v1.json"
)


checkpoint_tmp_restart0 = (
    checkpoint_dir_restart0
    /
    "fd003_post_629bc_restart_checkpoint_v1.json.tmp"
)


checkpoint_sha_path_restart0 = (
    checkpoint_dir_restart0
    /
    "fd003_post_629bc_restart_checkpoint_v1.json.sha256"
)


with open(
    checkpoint_tmp_restart0,
    "w",
    encoding="utf-8",
    newline="\n",
) as f_restart0:

    json.dump(
        checkpoint_wrapper_restart0,
        f_restart0,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )

    f_restart0.write("\n")


os.replace(
    checkpoint_tmp_restart0,
    checkpoint_path_restart0,
)


# =============================================================================
# 6) Compute whole-file SHA-256
# =============================================================================

with open(
    checkpoint_path_restart0,
    "rb",
) as f_restart0:

    checkpoint_file_bytes_restart0 = (
        f_restart0.read()
    )


checkpoint_file_sha256_restart0 = hashlib.sha256(
    checkpoint_file_bytes_restart0
).hexdigest()


with open(
    checkpoint_sha_path_restart0,
    "w",
    encoding="utf-8",
    newline="\n",
) as f_restart0:

    f_restart0.write(
        checkpoint_file_sha256_restart0
        + "  "
        + checkpoint_path_restart0.name
        + "\n"
    )


# =============================================================================
# 7) Immediate read-back verification
# =============================================================================

with open(
    checkpoint_path_restart0,
    "r",
    encoding="utf-8",
) as f_restart0:

    checkpoint_readback_restart0 = json.load(
        f_restart0
    )


if (
    checkpoint_readback_restart0.get(
        "payload_sha256"
    )
    !=
    checkpoint_payload_sha256_restart0
):
    raise RuntimeError(
        "Checkpoint read-back payload hash field mismatch."
    )


observed_payload_sha256_restart0 = _sha_json_restart0(
    checkpoint_readback_restart0[
        "payload"
    ]
)


if (
    observed_payload_sha256_restart0
    !=
    checkpoint_payload_sha256_restart0
):
    raise RuntimeError(
        "Checkpoint read-back payload fingerprint mismatch."
    )


readback_status_restart0 = (
    checkpoint_readback_restart0[
        "payload"
    ][
        "frozen_objects"
    ][
        "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC"
    ][
        "status"
    ]
)


if (
    readback_status_restart0
    !=
    "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED"
):
    raise RuntimeError(
        "Checkpoint read-back FD003 closure status mismatch."
    )


# =============================================================================
# 8) Current-session checkpoint state
# =============================================================================

FD003_RESTART_CHECKPOINT_CREATED_RESTART0 = True
FD003_RESTART_CHECKPOINT_VERIFIED_RESTART0 = True

FD003_RESTART_CHECKPOINT_PATH_RESTART0 = str(
    checkpoint_path_restart0
)

FD003_RESTART_CHECKPOINT_PAYLOAD_SHA256_RESTART0 = (
    checkpoint_payload_sha256_restart0
)

FD003_RESTART_CHECKPOINT_FILE_SHA256_RESTART0 = (
    checkpoint_file_sha256_restart0
)

all_checks_passed_restart0 = True


# =============================================================================
# 9) Output
# =============================================================================

print("=" * 166)

print(
    "STEP RESTART-0 — FD003 POST-CLOSURE RESTART-SAFE CHECKPOINT"
)

print("=" * 166)

print("FD003 final closure verified    :", True)
print("Checkpoint created              :", True)
print("Checkpoint read-back verified   :", True)
print("FD003 retraining performed      :", False)
print("FD003 threshold/policy changed  :", False)
print("TEST/RUL read again             :", False)

print("\n--- CHECKPOINT LOCATION ---")

print(
    "Checkpoint JSON                :",
    FD003_RESTART_CHECKPOINT_PATH_RESTART0,
)

print(
    "SHA-256 sidecar                :",
    str(checkpoint_sha_path_restart0),
)

print("\n--- FINGERPRINTS ---")

print(
    "Checkpoint payload SHA-256     :",
    FD003_RESTART_CHECKPOINT_PAYLOAD_SHA256_RESTART0,
)

print(
    "Checkpoint file SHA-256        :",
    FD003_RESTART_CHECKPOINT_FILE_SHA256_RESTART0,
)

print("\n--- RESUME CONTRACT ---")

print("Current completed dataset       : FD003")

print(
    "FD003 status                    : "
    "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED"
)

print("Next dataset                    : FD004")
print("Next notebook step after restore: STEP 6.30A")
print("FD004 numerical inheritance     : False")

print(
    "\nPASS — the essential frozen FD003 closure state has been "
    "written to disk and verified by immediate read-back."
)

print(
    "After saving this notebook, the laptop may be restarted. "
    "Do NOT run STEP 6.30A immediately after restart; first run "
    "the dedicated restore/verification cell."
)

In [ ]:
# STEP RESTART-1
# POST-RESTART FD003 CHECKPOINT RESTORE + RESUME VERIFICATION

import hashlib
import json
from pathlib import Path


# =============================================================================
# 1) Fixed checkpoint authority from pre-restart verification
# =============================================================================

checkpoint_path_restart1 = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
) / "fd003_post_629bc_restart_checkpoint_v1.json"

checkpoint_sha_path_restart1 = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
) / "fd003_post_629bc_restart_checkpoint_v1.json.sha256"

EXPECTED_CHECKPOINT_PAYLOAD_SHA256_RESTART1 = (
    "bd6206a6cb00f75c8e0be6893626580ba5c4b0483003b73da4ae2f8e9f605563"
)

EXPECTED_CHECKPOINT_FILE_SHA256_RESTART1 = (
    "b8f6fdf53a9aef48c78b13a38d76559de95710e46167ec951ea5a6024bf7aa5c"
)


# =============================================================================
# 2) Canonical hashing helper
# =============================================================================

def _sha_json_restart1(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Verify checkpoint files exist
# =============================================================================

if not checkpoint_path_restart1.is_file():
    raise RuntimeError(
        "Restart checkpoint JSON was not found at the expected path."
    )

if not checkpoint_sha_path_restart1.is_file():
    raise RuntimeError(
        "Restart checkpoint SHA-256 sidecar was not found."
    )


# =============================================================================
# 4) Verify whole-file SHA-256 BEFORE parsing
# =============================================================================

checkpoint_bytes_restart1 = checkpoint_path_restart1.read_bytes()

observed_file_sha256_restart1 = hashlib.sha256(
    checkpoint_bytes_restart1
).hexdigest()

if (
    observed_file_sha256_restart1
    != EXPECTED_CHECKPOINT_FILE_SHA256_RESTART1
):
    raise RuntimeError(
        "Restart checkpoint file SHA-256 does not match the "
        "pre-restart verified fingerprint."
    )

sidecar_text_restart1 = checkpoint_sha_path_restart1.read_text(
    encoding="utf-8"
).strip()

sidecar_hash_restart1 = sidecar_text_restart1.split()[0]

if sidecar_hash_restart1 != EXPECTED_CHECKPOINT_FILE_SHA256_RESTART1:
    raise RuntimeError(
        "Restart checkpoint SHA-256 sidecar does not match "
        "the pre-restart verified fingerprint."
    )


# =============================================================================
# 5) Parse and verify payload fingerprint
# =============================================================================

checkpoint_wrapper_restart1 = json.loads(
    checkpoint_bytes_restart1.decode("utf-8")
)

if set(checkpoint_wrapper_restart1.keys()) != {
    "payload",
    "payload_sha256",
}:
    raise RuntimeError(
        "Unexpected restart-checkpoint wrapper structure."
    )

checkpoint_payload_restart1 = checkpoint_wrapper_restart1["payload"]

stored_payload_sha256_restart1 = checkpoint_wrapper_restart1[
    "payload_sha256"
]

if (
    stored_payload_sha256_restart1
    != EXPECTED_CHECKPOINT_PAYLOAD_SHA256_RESTART1
):
    raise RuntimeError(
        "Stored checkpoint payload fingerprint differs from "
        "the pre-restart verified fingerprint."
    )

observed_payload_sha256_restart1 = _sha_json_restart1(
    checkpoint_payload_restart1
)

if (
    observed_payload_sha256_restart1
    != EXPECTED_CHECKPOINT_PAYLOAD_SHA256_RESTART1
):
    raise RuntimeError(
        "Recomputed checkpoint payload fingerprint mismatch."
    )


# =============================================================================
# 6) Verify resume semantics before restoring notebook state
# =============================================================================

if (
    checkpoint_payload_restart1.get("checkpoint_schema")
    != "FD003_POST_629BC_RESTART_CHECKPOINT_V1"
):
    raise RuntimeError(
        "Unexpected restart checkpoint schema."
    )

if (
    checkpoint_payload_restart1.get("resume_point")
    != "FD004_STEP_6.30A_PRE_ENTRY"
):
    raise RuntimeError(
        "Checkpoint resume point is not FD004 STEP 6.30A pre-entry."
    )

if (
    checkpoint_payload_restart1.get("fd003_status")
    != "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED"
):
    raise RuntimeError(
        "Checkpoint FD003 status is not the frozen closed status."
    )

if not bool(
    checkpoint_payload_restart1.get("fd003_closed", False)
):
    raise RuntimeError(
        "Checkpoint does not certify FD003 as closed."
    )

if checkpoint_payload_restart1.get("next_dataset") != "FD004":
    raise RuntimeError(
        "Checkpoint next dataset is not FD004."
    )

scientific_rules_restart1 = checkpoint_payload_restart1[
    "scientific_rules_after_restart"
]

required_true_rules_restart1 = [
    "do_not_retrain_fd003",
    "do_not_retune_fd003",
    "do_not_change_fd003_preprocessing",
    "do_not_change_fd003_risk_architecture",
    "do_not_change_fd003_threshold",
    "do_not_change_fd003_policy",
    "fd003_test_results_reporting_only",
    "fd004_inherit_methodology_only",
]

for rule_name_restart1 in required_true_rules_restart1:
    if not bool(
        scientific_rules_restart1.get(
            rule_name_restart1,
            False,
        )
    ):
        raise RuntimeError(
            "Checkpoint scientific rule is not locked True: "
            + rule_name_restart1
        )

if bool(
    scientific_rules_restart1.get(
        "fd004_inherit_fd003_numeric_state",
        True,
    )
):
    raise RuntimeError(
        "Checkpoint incorrectly permits FD003 numeric-state inheritance into FD004."
    )


# =============================================================================
# 7) Restore explicit frozen objects and hashes
# =============================================================================

frozen_restart1 = checkpoint_payload_restart1[
    "frozen_objects"
]

hashes_restart1 = checkpoint_payload_restart1[
    "expected_hashes"
]

FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC = frozen_restart1[
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC"
]

FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC = frozen_restart1[
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC"
]

FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB = frozen_restart1[
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB"
]

FD003_TEST_FEATURE_SIDE_CONTRACT_629BA = frozen_restart1[
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA"
]

FD003_TEST_ADMISSION_CONTRACT_629AY = frozen_restart1[
    "FD003_TEST_ADMISSION_CONTRACT_629AY"
]

FD003_PRETEST_EVIDENCE_CLOSURE_629AX = frozen_restart1[
    "FD003_PRETEST_EVIDENCE_CLOSURE_629AX"
]

FD003_OPERATING_POLICY_CONTRACT_629AU = frozen_restart1[
    "FD003_OPERATING_POLICY_CONTRACT_629AU"
]

FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW = frozen_restart1[
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW"
]

FD003_DEV_STAGE_CONCLUSION_629AX = frozen_restart1[
    "FD003_DEV_STAGE_CONCLUSION_629AX"
]


FD003_FINAL_DATASET_CLOSURE_SHA256_629BC = hashes_restart1[
    "FD003_FINAL_DATASET_CLOSURE_SHA256_629BC"
]

FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC = hashes_restart1[
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC"
]

FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB = hashes_restart1[
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB"
]

FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA = hashes_restart1[
    "FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA"
]

FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY = hashes_restart1[
    "FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY"
]

FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX = hashes_restart1[
    "FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX"
]

FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU = hashes_restart1[
    "FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU"
]

FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW = hashes_restart1[
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW"
]

FD003_DEV_STAGE_CONCLUSION_SHA256_629AX = hashes_restart1[
    "FD003_DEV_STAGE_CONCLUSION_SHA256_629AX"
]


# =============================================================================
# 8) Re-certify every restored frozen object
# =============================================================================

restored_pairs_restart1 = [
    (
        FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,
        FD003_FINAL_DATASET_CLOSURE_SHA256_629BC,
        "FD003 final dataset closure",
    ),
    (
        FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,
        FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,
        "FD003 final scientific conclusion",
    ),
    (
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,
        FD003_HELDOUT_TEST_SCORE_CONTRACT_SHA256_629BB,
        "FD003 held-out TEST score contract",
    ),
    (
        FD003_TEST_FEATURE_SIDE_CONTRACT_629BA,
        FD003_TEST_FEATURE_SIDE_CONTRACT_SHA256_629BA,
        "FD003 TEST feature-side contract",
    ),
    (
        FD003_TEST_ADMISSION_CONTRACT_629AY,
        FD003_TEST_ADMISSION_CONTRACT_SHA256_629AY,
        "FD003 TEST admission contract",
    ),
    (
        FD003_PRETEST_EVIDENCE_CLOSURE_629AX,
        FD003_PRETEST_EVIDENCE_CLOSURE_SHA256_629AX,
        "FD003 pre-TEST evidence closure",
    ),
    (
        FD003_OPERATING_POLICY_CONTRACT_629AU,
        FD003_OPERATING_POLICY_CONTRACT_SHA256_629AU,
        "FD003 operating policy contract",
    ),
    (
        FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW,
        FD003_FIXED_POLICY_REPORTING_CONTRACT_SHA256_629AW,
        "FD003 fixed-policy reporting contract",
    ),
    (
        FD003_DEV_STAGE_CONCLUSION_629AX,
        FD003_DEV_STAGE_CONCLUSION_SHA256_629AX,
        "FD003 DEV-stage conclusion",
    ),
]

for (
    restored_object_restart1,
    expected_object_sha_restart1,
    restored_label_restart1,
) in restored_pairs_restart1:

    if (
        _sha_json_restart1(
            restored_object_restart1
        )
        != expected_object_sha_restart1
    ):
        raise RuntimeError(
            restored_label_restart1
            + " failed post-restart fingerprint verification."
        )


# =============================================================================
# 9) Restore explicit closure flags
# =============================================================================

restore_flags_restart1 = checkpoint_payload_restart1[
    "restore_flags"
]

all_checks_passed_629bc = bool(
    restore_flags_restart1[
        "all_checks_passed_629bc"
    ]
)

FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC = bool(
    restore_flags_restart1[
        "FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC"
    ]
)

FD003_COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED_629BC = bool(
    restore_flags_restart1[
        "FD003_COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED_629BC"
    ]
)

FD003_CLOSED_629BC = bool(
    restore_flags_restart1[
        "FD003_CLOSED_629BC"
    ]
)

FD003_MODEL_RETRAINED_AFTER_TEST_629BC = bool(
    restore_flags_restart1[
        "FD003_MODEL_RETRAINED_AFTER_TEST_629BC"
    ]
)

FD003_PREPROCESSING_MODIFIED_AFTER_TEST_629BC = bool(
    restore_flags_restart1[
        "FD003_PREPROCESSING_MODIFIED_AFTER_TEST_629BC"
    ]
)

FD003_RISK_ARCHITECTURE_MODIFIED_AFTER_TEST_629BC = bool(
    restore_flags_restart1[
        "FD003_RISK_ARCHITECTURE_MODIFIED_AFTER_TEST_629BC"
    ]
)

FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC = bool(
    restore_flags_restart1[
        "FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC"
    ]
)

FD003_POLICY_MODIFIED_AFTER_TEST_629BC = bool(
    restore_flags_restart1[
        "FD003_POLICY_MODIFIED_AFTER_TEST_629BC"
    ]
)

FD003_TEST_RESULTS_USED_FOR_TUNING_629BC = bool(
    restore_flags_restart1[
        "FD003_TEST_RESULTS_USED_FOR_TUNING_629BC"
    ]
)


# =============================================================================
# 10) Final closure-state assertions
# =============================================================================

if not all_checks_passed_629bc:
    raise RuntimeError(
        "Restored all_checks_passed_629bc is not True."
    )

if not FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC:
    raise RuntimeError(
        "Restored FD003 final closure lock is not True."
    )

if not FD003_CLOSED_629BC:
    raise RuntimeError(
        "Restored FD003 closed flag is not True."
    )

if any(
    [
        FD003_MODEL_RETRAINED_AFTER_TEST_629BC,
        FD003_PREPROCESSING_MODIFIED_AFTER_TEST_629BC,
        FD003_RISK_ARCHITECTURE_MODIFIED_AFTER_TEST_629BC,
        FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC,
        FD003_POLICY_MODIFIED_AFTER_TEST_629BC,
        FD003_TEST_RESULTS_USED_FOR_TUNING_629BC,
    ]
):
    raise RuntimeError(
        "One or more restored post-TEST immutability flags are invalid."
    )

if (
    FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC[
        "next_dataset"
    ]
    != "FD004"
):
    raise RuntimeError(
        "Restored FD003 closure does not point to FD004."
    )


# =============================================================================
# 11) Post-restart resume state
# =============================================================================

FD003_RESTART_CHECKPOINT_RESTORED_RESTART1 = True
FD003_RESTART_CHECKPOINT_VERIFIED_RESTART1 = True

FD004_RESUME_AUTHORIZED_RESTART1 = True
FD004_NUMERIC_INHERITANCE_FROM_FD003_RESTART1 = False

all_checks_passed_restart1 = True


# =============================================================================
# 12) Output
# =============================================================================

print("=" * 168)

print(
    "STEP RESTART-1 — POST-RESTART FD003 CHECKPOINT RESTORE "
    "+ RESUME VERIFICATION"
)

print("=" * 168)

print("Checkpoint JSON found           :", True)
print("Checkpoint sidecar found        :", True)
print("Whole-file SHA-256 exact        :", True)
print("Payload SHA-256 exact           :", True)
print("Frozen FD003 objects restored   :", True)
print("All restored fingerprints exact :", True)

print("\n--- RESTORED PROJECT STATE ---")

print(
    "FD003 status                    :",
    FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC[
        "status"
    ],
)

print(
    "FD003 permanently closed        :",
    FD003_CLOSED_629BC,
)

print(
    "TEST results used for tuning    :",
    FD003_TEST_RESULTS_USED_FOR_TUNING_629BC,
)

print(
    "Post-TEST threshold modified    :",
    FD003_THRESHOLD_MODIFIED_AFTER_TEST_629BC,
)

print(
    "Post-TEST policy modified       :",
    FD003_POLICY_MODIFIED_AFTER_TEST_629BC,
)

print("\n--- RESTART AUTHORITY ---")

print(
    "Checkpoint payload SHA-256      :",
    observed_payload_sha256_restart1,
)

print(
    "Checkpoint file SHA-256         :",
    observed_file_sha256_restart1,
)

print(
    "Resume point                    :",
    checkpoint_payload_restart1[
        "resume_point"
    ],
)

print(
    "Next dataset                    :",
    checkpoint_payload_restart1[
        "next_dataset"
    ],
)

print(
    "FD003 numeric state -> FD004     :",
    FD004_NUMERIC_INHERITANCE_FROM_FD003_RESTART1,
)

print(
    "FD004 resume authorized         :",
    FD004_RESUME_AUTHORIZED_RESTART1,
)

print(
    "\nPASS — the post-restart notebook state is restored from the "
    "pre-restart verified checkpoint. FD003 remains permanently closed "
    "and no completed experiment was rerun."
)

print(
    "The notebook is now legitimately positioned at FD004 STEP 6.30A. "
    "Do not rerun any FD003 training, corruption, TEST, or RUL cells."
)

In [ ]:
# STEP 6.30A
# FD004 DATASET ENTRY + NO-LEAKAGE SOURCE-PATH LOCK

import hashlib
import json
import os


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_restart1", False)):
    raise RuntimeError(
        "STEP RESTART-1 must pass before FD004 entry."
    )

if not bool(
    globals().get(
        "FD004_RESUME_AUTHORIZED_RESTART1",
        False,
    )
):
    raise RuntimeError(
        "Post-restart authority does not permit FD004 entry."
    )

if bool(
    globals().get(
        "FD004_NUMERIC_INHERITANCE_FROM_FD003_RESTART1",
        True,
    )
):
    raise RuntimeError(
        "FD003 numerical state inheritance into FD004 is not permitted."
    )

required_630a = [
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",
    "FD003_FINAL_DATASET_CLOSURE_SHA256_629BC",
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC",
]

missing_630a = [
    name
    for name in required_630a
    if name not in globals()
]

if missing_630a:
    raise RuntimeError(
        "Missing restored FD003 closure object(s): "
        + ", ".join(missing_630a)
    )

if not bool(
    globals().get(
        "FD003_FINAL_DATASET_CLOSURE_LOCKED_629BC",
        False,
    )
):
    raise RuntimeError(
        "FD003 final closure is not locked."
    )

if not bool(
    globals().get(
        "FD003_CLOSED_629BC",
        False,
    )
):
    raise RuntimeError(
        "FD003 is not marked permanently closed."
    )

if bool(
    globals().get(
        "FD004_ENTRY_CONTRACT_LOCKED_630A",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30A has already been completed."
    )


# =============================================================================
# 2) Canonical helper
# =============================================================================

def _sha_json_630a(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 3) Re-certify FD003 closure before beginning FD004
# =============================================================================

if (
    _sha_json_630a(
        FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC
    )
    !=
    FD003_FINAL_DATASET_CLOSURE_SHA256_629BC
):
    raise RuntimeError(
        "Restored FD003 final closure contract changed before FD004 entry."
    )

if (
    FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC[
        "status"
    ]
    !=
    "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED"
):
    raise RuntimeError(
        "FD003 final status is not the expected closed status."
    )

if (
    FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC[
        "next_dataset"
    ]
    !=
    "FD004"
):
    raise RuntimeError(
        "FD003 closure contract does not authorize FD004 as next dataset."
    )


# =============================================================================
# 4) Freeze FD004 source paths WITHOUT reading contents
# =============================================================================

FD004_TRAIN_SOURCE_PATH_630A = (
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\train_FD004.txt"
)

FD004_TEST_SOURCE_PATH_630A = (
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\test_FD004.txt"
)

FD004_RUL_SOURCE_PATH_630A = (
    r"C:\RUL_Reliability_Research\02_dataset\raw\CMAPSSData\RUL_FD004.txt"
)

path_presence_630a = {
    "train_exists":
        bool(os.path.isfile(FD004_TRAIN_SOURCE_PATH_630A)),
    "test_exists":
        bool(os.path.isfile(FD004_TEST_SOURCE_PATH_630A)),
    "rul_exists":
        bool(os.path.isfile(FD004_RUL_SOURCE_PATH_630A)),
}

if not all(path_presence_630a.values()):
    missing_paths_630a = [
        key.replace("_exists", "")
        for key, present in path_presence_630a.items()
        if not present
    ]

    raise RuntimeError(
        "Expected FD004 source file(s) not found: "
        + ", ".join(missing_paths_630a)
    )


# =============================================================================
# 5) Freeze FD004 fresh-dataset entry contract
# =============================================================================

FD004_ENTRY_CONTRACT_630A = {

    "step":
        "6.30A",

    "dataset":
        "FD004",

    "research_problem":
        "Reliability-Aware Selective Remaining Useful Life Prediction "
        "under Sensor Degradation",

    "previous_dataset":
        "FD003",

    "previous_dataset_closure_sha256":
        FD003_FINAL_DATASET_CLOSURE_SHA256_629BC,

    "previous_dataset_final_conclusion_sha256":
        FD003_FINAL_SCIENTIFIC_CONCLUSION_SHA256_629BC,

    "train_source_path":
        FD004_TRAIN_SOURCE_PATH_630A,

    "test_source_path":
        FD004_TEST_SOURCE_PATH_630A,

    "official_rul_source_path":
        FD004_RUL_SOURCE_PATH_630A,

    "source_files_exist":
        path_presence_630a,

    "expected_train_engine_count":
        249,

    "expected_test_engine_count":
        248,

    "expected_operating_condition_count":
        6,

    "expected_fault_mode_count":
        2,

    "expected_raw_column_count":
        26,

    "expected_raw_schema":
        [
            "unit_id",
            "cycle",
            "op_setting_1",
            "op_setting_2",
            "op_setting_3",
            "sensor_1",
            "sensor_2",
            "sensor_3",
            "sensor_4",
            "sensor_5",
            "sensor_6",
            "sensor_7",
            "sensor_8",
            "sensor_9",
            "sensor_10",
            "sensor_11",
            "sensor_12",
            "sensor_13",
            "sensor_14",
            "sensor_15",
            "sensor_16",
            "sensor_17",
            "sensor_18",
            "sensor_19",
            "sensor_20",
            "sensor_21",
        ],

    "methodology_transfer_rule":
        (
            "Carry forward only the established research methodology, "
            "architecture family, leakage controls, and prospective-freeze "
            "principles. All FD004 dataset-specific preprocessing statistics, "
            "phase references, G1/G2 references, corruption references, "
            "thresholds, and held-out results must be created prospectively "
            "from FD004-permitted data."
        ),

    "fd003_numeric_threshold_inherited":
        False,

    "fd003_preprocessing_statistics_inherited":
        False,

    "fd003_phase_edges_inherited":
        False,

    "fd003_g1_reference_inherited":
        False,

    "fd003_g2_reference_inherited":
        False,

    "fd003_corruption_scale_reference_inherited":
        False,

    "fd003_dropout_imputation_reference_inherited":
        False,

    "fd003_test_results_used_for_fd004_selection":
        False,

    "train_contents_read":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "next_permitted_action":
        "TRAIN-only raw structural verification",
}


FD004_ENTRY_CONTRACT_SHA256_630A = _sha_json_630a(
    FD004_ENTRY_CONTRACT_630A
)


# =============================================================================
# 6) Lock state
# =============================================================================

FD004_ENTRY_CONTRACT_LOCKED_630A = True
FD004_DATASET_PHASE_STARTED_630A = True

FD004_TRAIN_CONTENTS_READ_630A = False
FD004_TEST_CONTENTS_READ_630A = False
FD004_RUL_CONTENTS_READ_630A = False

FD004_MODEL_TRAINING_PERFORMED_630A = False
FD004_THRESHOLD_SELECTED_630A = False
FD004_POLICY_SELECTED_630A = False

FD004_INHERITED_FD003_NUMERIC_STATE_630A = False

all_checks_passed_630a = True


# =============================================================================
# 7) Output
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30A — FD004 DATASET ENTRY + NO-LEAKAGE SOURCE-PATH LOCK"
)

print("=" * 168)

print("FD003 permanently closed       :", True)
print("FD004 phase started            :", True)

print("\n--- SOURCE PATH LOCK ---")

print(
    "TRAIN path                     :",
    FD004_TRAIN_SOURCE_PATH_630A,
)

print(
    "TEST path                      :",
    FD004_TEST_SOURCE_PATH_630A,
)

print(
    "Official RUL path              :",
    FD004_RUL_SOURCE_PATH_630A,
)

print(
    "Source files exist             :",
    path_presence_630a,
)

print("\n--- EXPECTED FD004 STRUCTURE (NOT YET READ) ---")

print("Expected TRAIN engines         :", 249)
print("Expected TEST engines          :", 248)
print("Expected operating conditions  :", 6)
print("Expected fault modes           :", 2)
print("Expected raw columns           :", 26)

print("\n--- NO-LEAKAGE STATE ---")

print("TRAIN contents read            :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)
print("Model training performed       :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)

print(
    "FD003 numerical state inherited:",
    False,
)

print(
    "\nFD004 entry-contract SHA-256   :",
    FD004_ENTRY_CONTRACT_SHA256_630A,
)

print(
    "\nPASS — FD004 has been entered as a fresh dataset phase with source "
    "paths locked and no TRAIN/TEST/RUL contents read."
)

print(
    "Only the established methodology is carried forward. No FD003 numerical "
    "threshold, calibration reference, preprocessing statistic, corruption "
    "reference, or held-out result is inherited."
)

print(
    "Next: read TRAIN_FD004.txt only and perform strict raw structural "
    "verification while keeping TEST and official RUL sealed."
)

In [ ]:
# STEP 6.30B
# FD004 TRAIN-ONLY RAW STRUCTURAL VERIFICATION + AUTHORITY LOCK
#
# Reads TRAIN_FD004.txt ONLY.
#
# NO TEST access.
# NO official RUL access.
# NO RUL labels constructed.
# NO FIT/DEV split.
# NO feature selection / normalization.
# NO windowing.
# NO model training.
# NO threshold / policy selection.

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630a", False)):
    raise RuntimeError(
        "STEP 6.30A must pass before STEP 6.30B."
    )

required_630b = [
    "FD004_ENTRY_CONTRACT_630A",
    "FD004_ENTRY_CONTRACT_SHA256_630A",
    "FD004_TRAIN_SOURCE_PATH_630A",
    "FD004_TEST_SOURCE_PATH_630A",
    "FD004_RUL_SOURCE_PATH_630A",
]

missing_630b = [
    name for name in required_630b
    if name not in globals()
]

if missing_630b:
    raise RuntimeError(
        "Missing STEP 6.30A object(s): "
        + ", ".join(missing_630b)
    )

if not bool(
    globals().get(
        "FD004_ENTRY_CONTRACT_LOCKED_630A",
        False,
    )
):
    raise RuntimeError(
        "FD004 entry contract is not locked."
    )

if bool(globals().get("FD004_TRAIN_CONTENTS_READ_630A", True)):
    raise RuntimeError(
        "TRAIN was unexpectedly marked read before STEP 6.30B."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630A", True)):
    raise RuntimeError(
        "TEST was unexpectedly marked read before STEP 6.30B."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630A", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly marked read before STEP 6.30B."
    )

if bool(
    globals().get(
        "FD004_TRAIN_RAW_AUTHORITY_LOCKED_630B",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30B has already been completed."
    )


# =============================================================================
# 2) Canonical helpers
# =============================================================================

def _sha_json_630b(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _sha_arr_630b(value):
    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.30B hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


# =============================================================================
# 3) Re-certify frozen FD004 entry contract
# =============================================================================

if (
    _sha_json_630b(
        FD004_ENTRY_CONTRACT_630A
    )
    !=
    FD004_ENTRY_CONTRACT_SHA256_630A
):
    raise RuntimeError(
        "STEP 6.30A FD004 entry contract changed."
    )

entry_630b = FD004_ENTRY_CONTRACT_630A

if entry_630b["dataset"] != "FD004":
    raise RuntimeError(
        "Entry contract dataset is not FD004."
    )

if bool(
    entry_630b[
        "fd003_numeric_threshold_inherited"
    ]
):
    raise RuntimeError(
        "FD003 numerical threshold inheritance is not permitted."
    )

raw_schema_630b = list(
    entry_630b[
        "expected_raw_schema"
    ]
)

if len(raw_schema_630b) != 26:
    raise RuntimeError(
        "Frozen FD004 raw schema does not contain 26 columns."
    )


# =============================================================================
# 4) READ TRAIN_FD004 ONLY
# =============================================================================

train_raw_630b = pd.read_csv(
    FD004_TRAIN_SOURCE_PATH_630A,
    sep=r"\s+",
    header=None,
    names=raw_schema_630b,
    engine="python",
)

FD004_TRAIN_CONTENTS_READ_630B = True
FD004_TEST_CONTENTS_READ_630B = False
FD004_RUL_CONTENTS_READ_630B = False


# =============================================================================
# 5) Strict raw structural checks
# =============================================================================

if train_raw_630b.shape[1] != 26:
    raise RuntimeError(
        "FD004 TRAIN raw column count is not 26."
    )

if list(train_raw_630b.columns) != raw_schema_630b:
    raise RuntimeError(
        "FD004 TRAIN column order differs from frozen schema."
    )

if train_raw_630b.shape[0] != 61249:
    raise RuntimeError(
        "FD004 TRAIN row count is not the expected 61249."
    )

raw_numeric_630b = train_raw_630b.to_numpy(
    dtype=np.float64,
    copy=True,
)

if raw_numeric_630b.shape != (61249, 26):
    raise RuntimeError(
        "FD004 TRAIN numeric matrix shape changed."
    )

if not np.isfinite(
    raw_numeric_630b
).all():
    raise RuntimeError(
        "FD004 TRAIN contains non-finite values."
    )

unit_float_630b = train_raw_630b[
    "unit_id"
].to_numpy(
    dtype=np.float64
)

cycle_float_630b = train_raw_630b[
    "cycle"
].to_numpy(
    dtype=np.float64
)

if not np.array_equal(
    unit_float_630b,
    unit_float_630b.astype(np.int64),
):
    raise RuntimeError(
        "FD004 TRAIN unit_id contains non-integer values."
    )

if not np.array_equal(
    cycle_float_630b,
    cycle_float_630b.astype(np.int64),
):
    raise RuntimeError(
        "FD004 TRAIN cycle contains non-integer values."
    )

train_raw_630b["unit_id"] = unit_float_630b.astype(
    np.int64
)

train_raw_630b["cycle"] = cycle_float_630b.astype(
    np.int64
)


# =============================================================================
# 6) Engine identity / ordering / cycle-contiguity checks
# =============================================================================

expected_train_engine_ids_630b = np.arange(
    1,
    250,
    dtype=np.int64,
)

observed_engine_ids_630b = (
    train_raw_630b[
        "unit_id"
    ]
    .drop_duplicates()
    .to_numpy(
        dtype=np.int64
    )
)

if not np.array_equal(
    observed_engine_ids_630b,
    expected_train_engine_ids_630b,
):
    raise RuntimeError(
        "FD004 TRAIN engine IDs are not exactly ordered 1..249."
    )

if int(
    train_raw_630b[
        "unit_id"
    ].nunique()
) != 249:
    raise RuntimeError(
        "FD004 TRAIN engine count is not exactly 249."
    )

if bool(
    train_raw_630b.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).any()
):
    raise RuntimeError(
        "Duplicate FD004 TRAIN (unit_id, cycle) rows detected."
    )

if bool(
    train_raw_630b.duplicated().any()
):
    raise RuntimeError(
        "Exact duplicate FD004 TRAIN rows detected."
    )

engine_lengths_630b = {}
engine_cycle_checks_630b = {}

for engine_id_630b in expected_train_engine_ids_630b:

    engine_rows_630b = train_raw_630b.loc[
        train_raw_630b[
            "unit_id"
        ]
        ==
        engine_id_630b
    ]

    cycles_630b = engine_rows_630b[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    )

    start_at_one_630b = bool(
        cycles_630b.size > 0
        and
        cycles_630b[0] == 1
    )

    contiguous_ordered_630b = bool(
        cycles_630b.size > 0
        and
        np.array_equal(
            cycles_630b,
            np.arange(
                1,
                cycles_630b[-1] + 1,
                dtype=np.int64,
            ),
        )
    )

    if not start_at_one_630b:
        raise RuntimeError(
            f"FD004 TRAIN engine {engine_id_630b} "
            "does not start at cycle 1."
        )

    if not contiguous_ordered_630b:
        raise RuntimeError(
            f"FD004 TRAIN engine {engine_id_630b} "
            "cycles are not contiguous and ordered."
        )

    engine_lengths_630b[
        int(engine_id_630b)
    ] = int(
        cycles_630b.size
    )

    engine_cycle_checks_630b[
        int(engine_id_630b)
    ] = {
        "start_at_one":
            start_at_one_630b,

        "contiguous_ordered":
            contiguous_ordered_630b,
    }


global_order_630b = train_raw_630b[
    [
        "unit_id",
        "cycle",
    ]
].to_numpy(
    dtype=np.int64,
    copy=True,
)

sorted_order_630b = (
    train_raw_630b[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .sort_values(
        [
            "unit_id",
            "cycle",
        ],
        kind="mergesort",
    )
    .to_numpy(
        dtype=np.int64,
        copy=True,
    )
)

global_unit_cycle_order_exact_630b = bool(
    np.array_equal(
        global_order_630b,
        sorted_order_630b,
    )
)

if not global_unit_cycle_order_exact_630b:
    raise RuntimeError(
        "FD004 TRAIN rows are not globally ordered by unit_id then cycle."
    )


# =============================================================================
# 7) Raw operating-setting evidence only
# =============================================================================

op_columns_630b = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

op_ranges_630b = {}
op_unique_counts_630b = {}

for op_col_630b in op_columns_630b:

    op_values_630b = train_raw_630b[
        op_col_630b
    ].to_numpy(
        dtype=np.float64
    )

    op_ranges_630b[
        op_col_630b
    ] = {
        "min":
            float(np.min(op_values_630b)),

        "max":
            float(np.max(op_values_630b)),
    }

    op_unique_counts_630b[
        op_col_630b
    ] = int(
        np.unique(op_values_630b).size
    )


# =============================================================================
# 8) Engine-length statistics
# =============================================================================

engine_length_vector_630b = np.asarray(
    [
        engine_lengths_630b[i]
        for i in range(
            1,
            250,
        )
    ],
    dtype=np.int64,
)

engine_length_stats_630b = {
    "min":
        int(engine_length_vector_630b.min()),

    "median":
        float(
            np.median(
                engine_length_vector_630b
            )
        ),

    "mean":
        float(
            np.mean(
                engine_length_vector_630b
            )
        ),

    "max":
        int(engine_length_vector_630b.max()),
}


# =============================================================================
# 9) Freeze authoritative TRAIN raw frame + fingerprints
# =============================================================================

FD004_TRAIN_RAW_630B = train_raw_630b.copy(
    deep=True
)

FD004_TRAIN_ENGINE_IDS_630B = observed_engine_ids_630b.copy()

FD004_TRAIN_ENGINE_LENGTHS_630B = engine_length_vector_630b.copy()

FD004_TRAIN_RAW_NUMERIC_SHA256_630B = _sha_arr_630b(
    FD004_TRAIN_RAW_630B[
        raw_schema_630b
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )
)

FD004_TRAIN_ENGINE_IDS_SHA256_630B = _sha_arr_630b(
    FD004_TRAIN_ENGINE_IDS_630B
)

FD004_TRAIN_ENGINE_LENGTHS_SHA256_630B = _sha_arr_630b(
    FD004_TRAIN_ENGINE_LENGTHS_630B
)


# =============================================================================
# 10) Freeze TRAIN-only raw-verification contract
# =============================================================================

FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B = {

    "step":
        "6.30B",

    "dataset":
        "FD004",

    "entry_contract_sha256":
        FD004_ENTRY_CONTRACT_SHA256_630A,

    "source_path":
        FD004_TRAIN_SOURCE_PATH_630A,

    "raw_shape":
        [61249, 26],

    "raw_schema":
        raw_schema_630b,

    "train_engine_count":
        249,

    "train_engine_ids":
        [
            int(x)
            for x in FD004_TRAIN_ENGINE_IDS_630B
        ],

    "engine_ids_exact_1_through_249":
        True,

    "duplicate_unit_cycle_rows":
        False,

    "exact_duplicate_rows":
        False,

    "all_numeric_values_finite":
        True,

    "global_unit_cycle_order_exact":
        True,

    "every_engine_starts_at_cycle_1":
        True,

    "every_engine_cycles_contiguous_ordered":
        True,

    "engine_length_stats":
        engine_length_stats_630b,

    "operating_setting_ranges":
        op_ranges_630b,

    "operating_setting_unique_counts":
        op_unique_counts_630b,

    "expected_operating_condition_count_from_dataset_definition":
        6,

    "operating_regime_assignment_performed":
        False,

    "fault_mode_labels_read_or_inferred":
        False,

    "train_raw_numeric_sha256":
        FD004_TRAIN_RAW_NUMERIC_SHA256_630B,

    "train_engine_ids_sha256":
        FD004_TRAIN_ENGINE_IDS_SHA256_630B,

    "train_engine_lengths_sha256":
        FD004_TRAIN_ENGINE_LENGTHS_SHA256_630B,

    "train_contents_read":
        True,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "rul_labels_constructed":
        False,

    "fit_dev_split_performed":
        False,

    "feature_selection_performed":
        False,

    "normalization_performed":
        False,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "fd003_numeric_state_inherited":
        False,
}


FD004_TRAIN_RAW_VERIFICATION_CONTRACT_SHA256_630B = (
    _sha_json_630b(
        FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B
    )
)


# =============================================================================
# 11) Lock state
# =============================================================================

FD004_TRAIN_ENGINE_IDS_630B.setflags(
    write=False
)

FD004_TRAIN_ENGINE_LENGTHS_630B.setflags(
    write=False
)

FD004_TRAIN_RAW_AUTHORITY_LOCKED_630B = True

FD004_TRAIN_CONTENTS_READ_630B = True
FD004_TEST_CONTENTS_READ_630B = False
FD004_RUL_CONTENTS_READ_630B = False

FD004_RUL_LABELS_CONSTRUCTED_630B = False
FD004_FIT_DEV_SPLIT_PERFORMED_630B = False
FD004_FEATURE_SELECTION_PERFORMED_630B = False
FD004_NORMALIZATION_PERFORMED_630B = False
FD004_WINDOWING_PERFORMED_630B = False
FD004_MODEL_TRAINING_PERFORMED_630B = False
FD004_THRESHOLD_SELECTED_630B = False
FD004_POLICY_SELECTED_630B = False

all_checks_passed_630b = True


# =============================================================================
# 12) Output
# =============================================================================

print("=" * 170)

print(
    "STEP 6.30B — FD004 TRAIN-ONLY RAW STRUCTURAL "
    "VERIFICATION + AUTHORITY LOCK"
)

print("=" * 170)

print("TRAIN contents read             :", True)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)
print("RUL labels constructed          :", False)
print("FIT/DEV split performed         :", False)
print("Feature selection performed     :", False)
print("Normalization performed         :", False)
print("Windowing performed             :", False)
print("Model training performed        :", False)
print("Threshold selected              :", False)
print("Policy selected                 :", False)

print("\n--- RAW TRAIN STRUCTURE ---")

print(
    "Raw shape                       :",
    FD004_TRAIN_RAW_630B.shape,
)

print(
    "Engine count                    :",
    len(
        FD004_TRAIN_ENGINE_IDS_630B
    ),
)

print(
    "Engine IDs exact 1..249         :",
    np.array_equal(
        FD004_TRAIN_ENGINE_IDS_630B,
        np.arange(
            1,
            250,
            dtype=np.int64,
        ),
    ),
)

print(
    "Duplicate (unit,cycle) rows     :",
    False,
)

print(
    "Exact duplicate rows            :",
    False,
)

print(
    "Global unit/cycle order exact   :",
    global_unit_cycle_order_exact_630b,
)

print(
    "All values finite               :",
    True,
)

print(
    "Engine cycle min/median/mean/max:",
    (
        engine_length_stats_630b["min"],
        engine_length_stats_630b["median"],
        engine_length_stats_630b["mean"],
        engine_length_stats_630b["max"],
    ),
)

print("\n--- OPERATING-SETTING RAW EVIDENCE ---")

for op_col_630b in op_columns_630b:
    print(
        f"{op_col_630b} range / unique count :",
        (
            op_ranges_630b[
                op_col_630b
            ][
                "min"
            ],
            op_ranges_630b[
                op_col_630b
            ][
                "max"
            ],
            op_unique_counts_630b[
                op_col_630b
            ],
        ),
    )

print(
    "Expected operating conditions   :",
    6,
)

print(
    "Operating regimes assigned now  :",
    False,
)

print(
    "Fault-mode labels read/inferred :",
    False,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "TRAIN raw numeric SHA-256       :",
    FD004_TRAIN_RAW_NUMERIC_SHA256_630B,
)

print(
    "TRAIN engine IDs SHA-256        :",
    FD004_TRAIN_ENGINE_IDS_SHA256_630B,
)

print(
    "TRAIN engine lengths SHA-256    :",
    FD004_TRAIN_ENGINE_LENGTHS_SHA256_630B,
)

print(
    "Raw-verification contract SHA-256:",
    FD004_TRAIN_RAW_VERIFICATION_CONTRACT_SHA256_630B,
)

print(
    "\nPASS — TRAIN_FD004.txt alone has been structurally verified and "
    "locked as the authoritative FD004 TRAIN raw source."
)

print(
    "TEST and official RUL remain sealed. Next: freeze an engine-aware "
    "FD004 FIT/DEV split using TRAIN engine IDs only; no RUL labels or "
    "preprocessing statistics will be created in that split step."
)

In [ ]:
# STEP 6.30C
# FD004 ENGINE-AWARE FIT/DEV SPLIT FREEZE

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Helpers
# =============================================================================

def _sha_arr_630c(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.30C hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_630c(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630b", False)):
    raise RuntimeError(
        "STEP 6.30B must pass before STEP 6.30C."
    )

required_630c = [
    "FD004_TRAIN_RAW_630B",
    "FD004_TRAIN_RAW_NUMERIC_SHA256_630B",
    "FD004_TRAIN_ENGINE_IDS_630B",
    "FD004_TRAIN_ENGINE_IDS_SHA256_630B",
    "FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B",
    "FD004_TRAIN_RAW_VERIFICATION_CONTRACT_SHA256_630B",
]

missing_630c = [
    name for name in required_630c
    if name not in globals()
]

if missing_630c:
    raise RuntimeError(
        "Missing STEP 6.30B object(s): "
        + ", ".join(missing_630c)
    )

if not bool(
    globals().get(
        "FD004_TRAIN_RAW_AUTHORITY_LOCKED_630B",
        False,
    )
):
    raise RuntimeError(
        "FD004 TRAIN raw authority is not locked."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630B", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30C."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630B", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30C."
    )

if bool(
    globals().get(
        "FD004_FIT_DEV_SPLIT_LOCKED_630C",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30C has already been completed."
    )


# =============================================================================
# 3) Re-certify authoritative TRAIN evidence
# =============================================================================

if (
    _sha_json_630c(
        FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B
    )
    !=
    FD004_TRAIN_RAW_VERIFICATION_CONTRACT_SHA256_630B
):
    raise RuntimeError(
        "STEP 6.30B raw-verification contract changed."
    )

raw_schema_630c = list(
    FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B[
        "raw_schema"
    ]
)

if (
    _sha_arr_630c(
        FD004_TRAIN_RAW_630B[
            raw_schema_630c
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
    !=
    FD004_TRAIN_RAW_NUMERIC_SHA256_630B
):
    raise RuntimeError(
        "Authoritative FD004 TRAIN raw frame changed."
    )

if (
    _sha_arr_630c(
        FD004_TRAIN_ENGINE_IDS_630B
    )
    !=
    FD004_TRAIN_ENGINE_IDS_SHA256_630B
):
    raise RuntimeError(
        "Authoritative FD004 TRAIN engine IDs changed."
    )


# =============================================================================
# 4) Freeze prospective engine-level split rule
# =============================================================================

engine_ids_630c = np.asarray(
    FD004_TRAIN_ENGINE_IDS_630B,
    dtype=np.int64,
).reshape(-1)

if not np.array_equal(
    engine_ids_630c,
    np.arange(
        1,
        250,
        dtype=np.int64,
    ),
):
    raise RuntimeError(
        "FD004 TRAIN engine IDs are no longer exactly 1..249."
    )


FD004_SPLIT_SEED_630C = 63004
FD004_DEV_FRACTION_630C = 0.20

FD004_DEV_ENGINE_COUNT_630C = int(
    round(
        FD004_DEV_FRACTION_630C
        *
        engine_ids_630c.size
    )
)

FD004_FIT_ENGINE_COUNT_630C = int(
    engine_ids_630c.size
    -
    FD004_DEV_ENGINE_COUNT_630C
)

if FD004_DEV_ENGINE_COUNT_630C != 50:
    raise RuntimeError(
        "Prospective FD004 DEV engine count is not 50."
    )

if FD004_FIT_ENGINE_COUNT_630C != 199:
    raise RuntimeError(
        "Prospective FD004 FIT engine count is not 199."
    )


rng_630c = np.random.default_rng(
    FD004_SPLIT_SEED_630C
)

permuted_engine_ids_630c = rng_630c.permutation(
    engine_ids_630c
)


FD004_DEV_ENGINE_IDS_630C = np.sort(
    permuted_engine_ids_630c[
        :FD004_DEV_ENGINE_COUNT_630C
    ]
).astype(
    np.int64,
    copy=False,
)


FD004_FIT_ENGINE_IDS_630C = np.sort(
    permuted_engine_ids_630c[
        FD004_DEV_ENGINE_COUNT_630C:
    ]
).astype(
    np.int64,
    copy=False,
)


# =============================================================================
# 5) Exact partition checks
# =============================================================================

if np.intersect1d(
    FD004_FIT_ENGINE_IDS_630C,
    FD004_DEV_ENGINE_IDS_630C,
).size != 0:
    raise RuntimeError(
        "FD004 FIT/DEV engine overlap detected."
    )


combined_ids_630c = np.sort(
    np.concatenate(
        [
            FD004_FIT_ENGINE_IDS_630C,
            FD004_DEV_ENGINE_IDS_630C,
        ]
    )
)

if not np.array_equal(
    combined_ids_630c,
    engine_ids_630c,
):
    raise RuntimeError(
        "FD004 FIT/DEV engine union does not recover all TRAIN engines."
    )


if np.unique(
    FD004_FIT_ENGINE_IDS_630C
).size != 199:
    raise RuntimeError(
        "FD004 FIT engine IDs are not unique."
    )


if np.unique(
    FD004_DEV_ENGINE_IDS_630C
).size != 50:
    raise RuntimeError(
        "FD004 DEV engine IDs are not unique."
    )


# =============================================================================
# 6) Materialize raw FIT / DEV partitions
# =============================================================================

FD004_FIT_RAW_630C = FD004_TRAIN_RAW_630B.loc[
    FD004_TRAIN_RAW_630B[
        "unit_id"
    ].isin(
        FD004_FIT_ENGINE_IDS_630C
    )
].copy(
    deep=True
)


FD004_DEV_RAW_630C = FD004_TRAIN_RAW_630B.loc[
    FD004_TRAIN_RAW_630B[
        "unit_id"
    ].isin(
        FD004_DEV_ENGINE_IDS_630C
    )
].copy(
    deep=True
)


fit_row_ids_630c = np.unique(
    FD004_FIT_RAW_630C[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    )
)

dev_row_ids_630c = np.unique(
    FD004_DEV_RAW_630C[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    fit_row_ids_630c,
    FD004_FIT_ENGINE_IDS_630C,
):
    raise RuntimeError(
        "FD004 FIT raw-frame membership does not match frozen FIT IDs."
    )


if not np.array_equal(
    dev_row_ids_630c,
    FD004_DEV_ENGINE_IDS_630C,
):
    raise RuntimeError(
        "FD004 DEV raw-frame membership does not match frozen DEV IDs."
    )


if (
    FD004_FIT_RAW_630C.shape[0]
    +
    FD004_DEV_RAW_630C.shape[0]
    !=
    FD004_TRAIN_RAW_630B.shape[0]
):
    raise RuntimeError(
        "FD004 FIT/DEV row counts do not recover all TRAIN rows."
    )


# =============================================================================
# 7) Fingerprints
# =============================================================================

FD004_FIT_ENGINE_IDS_SHA256_630C = _sha_arr_630c(
    FD004_FIT_ENGINE_IDS_630C
)

FD004_DEV_ENGINE_IDS_SHA256_630C = _sha_arr_630c(
    FD004_DEV_ENGINE_IDS_630C
)

FD004_FIT_RAW_NUMERIC_SHA256_630C = _sha_arr_630c(
    FD004_FIT_RAW_630C[
        raw_schema_630c
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )
)

FD004_DEV_RAW_NUMERIC_SHA256_630C = _sha_arr_630c(
    FD004_DEV_RAW_630C[
        raw_schema_630c
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )
)


# =============================================================================
# 8) Freeze split contract
# =============================================================================

FD004_FIT_DEV_SPLIT_CONTRACT_630C = {

    "step":
        "6.30C",

    "dataset":
        "FD004",

    "raw_verification_contract_sha256":
        FD004_TRAIN_RAW_VERIFICATION_CONTRACT_SHA256_630B,

    "split_population":
        "TRAIN engine IDs only",

    "split_unit":
        "engine",

    "split_fraction_dev":
        FD004_DEV_FRACTION_630C,

    "split_seed":
        FD004_SPLIT_SEED_630C,

    "rng":
        "numpy.random.default_rng",

    "selection_rule":
        (
            "permute sorted TRAIN engine IDs with seed 63004; "
            "first 50 -> DEV; remainder -> FIT; "
            "store each partition sorted"
        ),

    "train_engine_count":
        249,

    "fit_engine_count":
        199,

    "dev_engine_count":
        50,

    "fit_dev_overlap_count":
        0,

    "fit_dev_union_exact":
        True,

    "fit_row_count":
        int(
            FD004_FIT_RAW_630C.shape[0]
        ),

    "dev_row_count":
        int(
            FD004_DEV_RAW_630C.shape[0]
        ),

    "train_row_count":
        int(
            FD004_TRAIN_RAW_630B.shape[0]
        ),

    "fit_engine_ids_sha256":
        FD004_FIT_ENGINE_IDS_SHA256_630C,

    "dev_engine_ids_sha256":
        FD004_DEV_ENGINE_IDS_SHA256_630C,

    "fit_raw_numeric_sha256":
        FD004_FIT_RAW_NUMERIC_SHA256_630C,

    "dev_raw_numeric_sha256":
        FD004_DEV_RAW_NUMERIC_SHA256_630C,

    "rul_labels_constructed":
        False,

    "operating_regime_assignment_used_for_split":
        False,

    "feature_selection_performed":
        False,

    "normalization_performed":
        False,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "fd003_numeric_state_inherited":
        False,
}


FD004_FIT_DEV_SPLIT_CONTRACT_SHA256_630C = _sha_json_630c(
    FD004_FIT_DEV_SPLIT_CONTRACT_630C
)


# =============================================================================
# 9) Lock state
# =============================================================================

FD004_FIT_ENGINE_IDS_630C.setflags(
    write=False
)

FD004_DEV_ENGINE_IDS_630C.setflags(
    write=False
)

FD004_FIT_DEV_SPLIT_PERFORMED_630C = True
FD004_FIT_DEV_SPLIT_LOCKED_630C = True

FD004_RUL_LABELS_CONSTRUCTED_630C = False
FD004_FEATURE_SELECTION_PERFORMED_630C = False
FD004_NORMALIZATION_PERFORMED_630C = False
FD004_WINDOWING_PERFORMED_630C = False
FD004_MODEL_TRAINING_PERFORMED_630C = False

FD004_TEST_CONTENTS_READ_630C = False
FD004_RUL_CONTENTS_READ_630C = False

FD004_THRESHOLD_SELECTED_630C = False
FD004_POLICY_SELECTED_630C = False

all_checks_passed_630c = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 170)

print(
    "STEP 6.30C — FD004 ENGINE-AWARE FIT/DEV SPLIT FREEZE"
)

print("=" * 170)

print("Split performed                 :", True)
print("Split unit                      : engine")
print("RUL labels constructed          :", False)
print("Operating regimes used in split :", False)
print("Feature selection performed     :", False)
print("Normalization performed         :", False)
print("Windowing performed             :", False)
print("Model training performed        :", False)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)
print("Threshold selected              :", False)
print("Policy selected                 :", False)

print("\n--- FROZEN SPLIT DESIGN ---")

print("TRAIN engines                   :", 249)
print("FIT engines                     :", 199)
print("DEV engines                     :", 50)
print("DEV fraction                    :", FD004_DEV_FRACTION_630C)
print("Split seed                      :", FD004_SPLIT_SEED_630C)
print("FIT / DEV overlap               :", 0)
print("FIT + DEV union exact           :", True)

print("\nDEV engine IDs:")

print(
    FD004_DEV_ENGINE_IDS_630C.tolist()
)

print("\n--- ROW PARTITIONS ---")

print(
    "FIT rows                        :",
    FD004_FIT_RAW_630C.shape[0],
)

print(
    "DEV rows                        :",
    FD004_DEV_RAW_630C.shape[0],
)

print(
    "FIT + DEV rows                 :",
    FD004_FIT_RAW_630C.shape[0]
    +
    FD004_DEV_RAW_630C.shape[0],
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "FIT engine IDs SHA-256         :",
    FD004_FIT_ENGINE_IDS_SHA256_630C,
)

print(
    "DEV engine IDs SHA-256         :",
    FD004_DEV_ENGINE_IDS_SHA256_630C,
)

print(
    "FIT raw numeric SHA-256        :",
    FD004_FIT_RAW_NUMERIC_SHA256_630C,
)

print(
    "DEV raw numeric SHA-256        :",
    FD004_DEV_RAW_NUMERIC_SHA256_630C,
)

print(
    "Split contract SHA-256         :",
    FD004_FIT_DEV_SPLIT_CONTRACT_SHA256_630C,
)

print(
    "\nPASS — the FD004 TRAIN population is now partitioned prospectively "
    "at engine level into disjoint FIT and DEV sets using engine IDs only."
)

print(
    "TEST and official RUL remain sealed. Next: construct uncapped "
    "TRAIN-derived RUL labels separately within the already-frozen FIT/DEV "
    "partition and audit FIT-only raw feature variability before any "
    "feature selection."
)

In [ ]:
# STEP 6.30D
# FD004 TRAIN-DERIVED UNCAPPED RUL LABELS + FIT-ONLY RAW VARIABILITY AUDIT

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 1) Helpers
# =============================================================================

def _sha_arr_630d(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.30D hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_630d(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630c", False)):
    raise RuntimeError(
        "STEP 6.30C must pass before STEP 6.30D."
    )

required_630d = [
    "FD004_FIT_RAW_630C",
    "FD004_DEV_RAW_630C",
    "FD004_FIT_ENGINE_IDS_630C",
    "FD004_DEV_ENGINE_IDS_630C",
    "FD004_FIT_RAW_NUMERIC_SHA256_630C",
    "FD004_DEV_RAW_NUMERIC_SHA256_630C",
    "FD004_FIT_DEV_SPLIT_CONTRACT_630C",
    "FD004_FIT_DEV_SPLIT_CONTRACT_SHA256_630C",
    "FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B",
]

missing_630d = [
    name for name in required_630d
    if name not in globals()
]

if missing_630d:
    raise RuntimeError(
        "Missing STEP 6.30C object(s): "
        + ", ".join(missing_630d)
    )

if not bool(
    globals().get(
        "FD004_FIT_DEV_SPLIT_LOCKED_630C",
        False,
    )
):
    raise RuntimeError(
        "FD004 FIT/DEV split is not locked."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630C", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30D."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630C", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30D."
    )

if bool(
    globals().get(
        "FD004_RUL_VARIABILITY_LOCKED_630D",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30D has already been completed."
    )


# =============================================================================
# 3) Re-certify frozen split
# =============================================================================

if (
    _sha_json_630d(
        FD004_FIT_DEV_SPLIT_CONTRACT_630C
    )
    !=
    FD004_FIT_DEV_SPLIT_CONTRACT_SHA256_630C
):
    raise RuntimeError(
        "STEP 6.30C split contract changed."
    )

raw_schema_630d = list(
    FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B[
        "raw_schema"
    ]
)

if (
    _sha_arr_630d(
        FD004_FIT_RAW_630C[
            raw_schema_630d
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
    !=
    FD004_FIT_RAW_NUMERIC_SHA256_630C
):
    raise RuntimeError(
        "Frozen FD004 FIT raw frame changed."
    )

if (
    _sha_arr_630d(
        FD004_DEV_RAW_630C[
            raw_schema_630d
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
    !=
    FD004_DEV_RAW_NUMERIC_SHA256_630C
):
    raise RuntimeError(
        "Frozen FD004 DEV raw frame changed."
    )


# =============================================================================
# 4) Construct uncapped RUL within frozen FIT / DEV partitions
# =============================================================================

fit_terminal_cycle_630d = (
    FD004_FIT_RAW_630C
    .groupby(
        "unit_id",
        sort=False,
    )[
        "cycle"
    ]
    .transform("max")
    .to_numpy(
        dtype=np.int64
    )
)

dev_terminal_cycle_630d = (
    FD004_DEV_RAW_630C
    .groupby(
        "unit_id",
        sort=False,
    )[
        "cycle"
    ]
    .transform("max")
    .to_numpy(
        dtype=np.int64
    )
)

fit_cycle_630d = FD004_FIT_RAW_630C[
    "cycle"
].to_numpy(
    dtype=np.int64
)

dev_cycle_630d = FD004_DEV_RAW_630C[
    "cycle"
].to_numpy(
    dtype=np.int64
)


FD004_FIT_RUL_630D = (
    fit_terminal_cycle_630d
    -
    fit_cycle_630d
).astype(
    np.int64,
    copy=False,
)

FD004_DEV_RUL_630D = (
    dev_terminal_cycle_630d
    -
    dev_cycle_630d
).astype(
    np.int64,
    copy=False,
)


if FD004_FIT_RUL_630D.shape != (
    FD004_FIT_RAW_630C.shape[0],
):
    raise RuntimeError(
        "FIT RUL vector shape mismatch."
    )

if FD004_DEV_RUL_630D.shape != (
    FD004_DEV_RAW_630C.shape[0],
):
    raise RuntimeError(
        "DEV RUL vector shape mismatch."
    )

if (
    (FD004_FIT_RUL_630D < 0).any()
    or
    (FD004_DEV_RUL_630D < 0).any()
):
    raise RuntimeError(
        "Negative TRAIN-derived RUL label detected."
    )


fit_zero_count_630d = int(
    np.count_nonzero(
        FD004_FIT_RUL_630D == 0
    )
)

dev_zero_count_630d = int(
    np.count_nonzero(
        FD004_DEV_RUL_630D == 0
    )
)

if fit_zero_count_630d != len(
    FD004_FIT_ENGINE_IDS_630C
):
    raise RuntimeError(
        "FIT zero-RUL endpoint count does not equal FIT engine count."
    )

if dev_zero_count_630d != len(
    FD004_DEV_ENGINE_IDS_630C
):
    raise RuntimeError(
        "DEV zero-RUL endpoint count does not equal DEV engine count."
    )


# =============================================================================
# 5) Per-engine target-alignment verification
# =============================================================================

for (
    frame_630d,
    rul_630d,
    ids_630d,
    label_630d,
) in (
    (
        FD004_FIT_RAW_630C,
        FD004_FIT_RUL_630D,
        FD004_FIT_ENGINE_IDS_630C,
        "FIT",
    ),
    (
        FD004_DEV_RAW_630C,
        FD004_DEV_RUL_630D,
        FD004_DEV_ENGINE_IDS_630C,
        "DEV",
    ),
):

    unit_vec_630d = frame_630d[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    )

    cycle_vec_630d = frame_630d[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    )

    for engine_id_630d in ids_630d:

        mask_630d = (
            unit_vec_630d
            ==
            int(engine_id_630d)
        )

        engine_rul_630d = rul_630d[
            mask_630d
        ]

        engine_cycle_630d = cycle_vec_630d[
            mask_630d
        ]

        if engine_rul_630d.size == 0:
            raise RuntimeError(
                f"{label_630d} engine {engine_id_630d} has no rows."
            )

        if int(
            engine_rul_630d[-1]
        ) != 0:
            raise RuntimeError(
                f"{label_630d} engine {engine_id_630d} "
                "endpoint RUL is not zero."
            )

        expected_rul_630d = (
            int(
                engine_cycle_630d[-1]
            )
            -
            engine_cycle_630d
        )

        if not np.array_equal(
            engine_rul_630d,
            expected_rul_630d,
        ):
            raise RuntimeError(
                f"{label_630d} engine {engine_id_630d} "
                "RUL alignment failed."
            )


# =============================================================================
# 6) FIT-only raw feature variability audit
# =============================================================================

raw_candidates_630d = [
    c
    for c in raw_schema_630d
    if c not in (
        "unit_id",
        "cycle",
    )
]

if len(raw_candidates_630d) != 24:
    raise RuntimeError(
        "Expected 24 raw candidate features before FIT-only audit."
    )


fit_variability_records_630d = []
excluded_constants_630d = []
retained_nonconstant_630d = []


for feature_630d in raw_candidates_630d:

    values_630d = FD004_FIT_RAW_630C[
        feature_630d
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )

    if not np.isfinite(
        values_630d
    ).all():
        raise RuntimeError(
            f"FIT feature {feature_630d} contains non-finite values."
        )

    exact_constant_630d = bool(
        values_630d.size > 0
        and
        np.all(
            values_630d
            ==
            values_630d[0]
        )
    )

    unique_count_630d = int(
        np.unique(
            values_630d
        ).size
    )

    min_630d = float(
        np.min(
            values_630d
        )
    )

    max_630d = float(
        np.max(
            values_630d
        )
    )

    fit_variability_records_630d.append(
        {
            "feature":
                feature_630d,

            "exact_constant":
                exact_constant_630d,

            "unique_count":
                unique_count_630d,

            "min":
                min_630d,

            "max":
                max_630d,

            "range":
                float(
                    max_630d
                    -
                    min_630d
                ),
        }
    )

    if exact_constant_630d:
        excluded_constants_630d.append(
            feature_630d
        )
    else:
        retained_nonconstant_630d.append(
            feature_630d
        )


if (
    len(excluded_constants_630d)
    +
    len(retained_nonconstant_630d)
    != 24
):
    raise RuntimeError(
        "FIT-only variability partition does not cover all 24 candidates."
    )


FD004_RAW_CANDIDATE_FEATURES_630D = list(
    raw_candidates_630d
)

FD004_FIT_EXACT_CONSTANT_FEATURES_630D = list(
    excluded_constants_630d
)

FD004_FIT_NONCONSTANT_FEATURES_630D = list(
    retained_nonconstant_630d
)

FD004_FIT_VARIABILITY_AUDIT_630D = pd.DataFrame(
    fit_variability_records_630d
)


# =============================================================================
# 7) Fingerprints
# =============================================================================

FD004_FIT_RUL_SHA256_630D = _sha_arr_630d(
    FD004_FIT_RUL_630D
)

FD004_DEV_RUL_SHA256_630D = _sha_arr_630d(
    FD004_DEV_RUL_630D
)

FD004_RAW_CANDIDATE_FEATURES_SHA256_630D = _sha_json_630d(
    FD004_RAW_CANDIDATE_FEATURES_630D
)

FD004_FIT_EXACT_CONSTANT_FEATURES_SHA256_630D = _sha_json_630d(
    FD004_FIT_EXACT_CONSTANT_FEATURES_630D
)

FD004_FIT_NONCONSTANT_FEATURES_SHA256_630D = _sha_json_630d(
    FD004_FIT_NONCONSTANT_FEATURES_630D
)

FD004_FIT_VARIABILITY_AUDIT_SHA256_630D = _sha_json_630d(
    fit_variability_records_630d
)


# =============================================================================
# 8) Freeze RUL / variability contract
# =============================================================================

FD004_RUL_VARIABILITY_CONTRACT_630D = {

    "step":
        "6.30D",

    "dataset":
        "FD004",

    "split_contract_sha256":
        FD004_FIT_DEV_SPLIT_CONTRACT_SHA256_630C,

    "rul_definition":
        "uncapped engine_terminal_cycle - current_cycle",

    "rul_source":
        "TRAIN trajectories only",

    "fit_rul_shape":
        [
            int(
                FD004_FIT_RUL_630D.size
            )
        ],

    "dev_rul_shape":
        [
            int(
                FD004_DEV_RUL_630D.size
            )
        ],

    "fit_rul_min":
        int(
            FD004_FIT_RUL_630D.min()
        ),

    "fit_rul_max":
        int(
            FD004_FIT_RUL_630D.max()
        ),

    "dev_rul_min":
        int(
            FD004_DEV_RUL_630D.min()
        ),

    "dev_rul_max":
        int(
            FD004_DEV_RUL_630D.max()
        ),

    "fit_zero_rul_endpoints":
        fit_zero_count_630d,

    "dev_zero_rul_endpoints":
        dev_zero_count_630d,

    "fit_rul_sha256":
        FD004_FIT_RUL_SHA256_630D,

    "dev_rul_sha256":
        FD004_DEV_RUL_SHA256_630D,

    "raw_candidate_feature_count":
        24,

    "raw_candidate_features":
        FD004_RAW_CANDIDATE_FEATURES_630D,

    "fit_only_variability_rule":
        "exclude only features exactly constant over FIT rows",

    "fit_exact_constant_feature_count":
        len(
            FD004_FIT_EXACT_CONSTANT_FEATURES_630D
        ),

    "fit_exact_constant_features":
        FD004_FIT_EXACT_CONSTANT_FEATURES_630D,

    "fit_nonconstant_feature_count":
        len(
            FD004_FIT_NONCONSTANT_FEATURES_630D
        ),

    "fit_nonconstant_features":
        FD004_FIT_NONCONSTANT_FEATURES_630D,

    "raw_candidate_features_sha256":
        FD004_RAW_CANDIDATE_FEATURES_SHA256_630D,

    "fit_exact_constant_features_sha256":
        FD004_FIT_EXACT_CONSTANT_FEATURES_SHA256_630D,

    "fit_nonconstant_features_sha256":
        FD004_FIT_NONCONSTANT_FEATURES_SHA256_630D,

    "fit_variability_audit_sha256":
        FD004_FIT_VARIABILITY_AUDIT_SHA256_630D,

    "dev_used_for_feature_variability_decision":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "normalization_performed":
        False,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "fd003_numeric_state_inherited":
        False,
}


FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D = _sha_json_630d(
    FD004_RUL_VARIABILITY_CONTRACT_630D
)


# =============================================================================
# 9) Lock state
# =============================================================================

FD004_FIT_RUL_630D.setflags(
    write=False
)

FD004_DEV_RUL_630D.setflags(
    write=False
)

FD004_RUL_LABELS_CONSTRUCTED_630D = True
FD004_FIT_VARIABILITY_AUDITED_630D = True
FD004_RUL_VARIABILITY_LOCKED_630D = True

FD004_TEST_CONTENTS_READ_630D = False
FD004_RUL_CONTENTS_READ_630D = False

FD004_NORMALIZATION_PERFORMED_630D = False
FD004_WINDOWING_PERFORMED_630D = False
FD004_MODEL_TRAINING_PERFORMED_630D = False
FD004_THRESHOLD_SELECTED_630D = False
FD004_POLICY_SELECTED_630D = False

all_checks_passed_630d = True


# =============================================================================
# 10) Output
# =============================================================================

print("=" * 170)

print(
    "STEP 6.30D — FD004 TRAIN-DERIVED UNCAPPED RUL LABELS "
    "+ FIT-ONLY RAW VARIABILITY AUDIT"
)

print("=" * 170)

print("RUL labels constructed          :", True)
print("RUL clipping/capping used       :", False)
print("Feature variability source      : FIT only")
print("DEV used for variability choice :", False)
print("Normalization performed         :", False)
print("Windowing performed             :", False)
print("Model training performed        :", False)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)
print("Threshold selected              :", False)
print("Policy selected                 :", False)

print("\n--- UNCAPPED TRAIN-DERIVED RUL ---")

print(
    "FIT RUL min / max               :",
    (
        int(
            FD004_FIT_RUL_630D.min()
        ),
        int(
            FD004_FIT_RUL_630D.max()
        ),
    ),
)

print(
    "DEV RUL min / max               :",
    (
        int(
            FD004_DEV_RUL_630D.min()
        ),
        int(
            FD004_DEV_RUL_630D.max()
        ),
    ),
)

print(
    "FIT / DEV zero-RUL endpoints    :",
    (
        fit_zero_count_630d,
        dev_zero_count_630d,
    ),
)

print("\n--- FIT-ONLY RAW VARIABILITY ---")

print(
    "Raw candidate feature count     :",
    24,
)

print(
    "Exact FIT constant features     :",
    FD004_FIT_EXACT_CONSTANT_FEATURES_630D,
)

print(
    "Nonconstant FIT feature count   :",
    len(
        FD004_FIT_NONCONSTANT_FEATURES_630D
    ),
)

print(
    "Nonconstant FIT features        :",
    FD004_FIT_NONCONSTANT_FEATURES_630D,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "FIT RUL SHA-256                 :",
    FD004_FIT_RUL_SHA256_630D,
)

print(
    "DEV RUL SHA-256                 :",
    FD004_DEV_RUL_SHA256_630D,
)

print(
    "FIT variability audit SHA-256   :",
    FD004_FIT_VARIABILITY_AUDIT_SHA256_630D,
)

print(
    "RUL/variability contract SHA-256:",
    FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D,
)

print(
    "\nPASS — uncapped TRAIN-derived RUL labels are now aligned within "
    "the already-frozen FIT/DEV split, and exact-constant filtering "
    "evidence was derived from FIT only."
)

print(
    "TEST and official RUL remain sealed. Next: freeze the FD004 model "
    "feature set from the FIT-only nonconstant list and fit normalization "
    "statistics on FIT only, then transform DEV without using DEV to fit "
    "the scaler."
)

In [ ]:
# STEP 6.30D-R1
# FD004 HISTORICAL MULTI-REGIME PREPROCESSING PROVENANCE LOCATOR
#
# SOURCE-ONLY FORENSIC STEP.
#
# Reads ONLY Jupyter notebook JSON/source text under the notebook directory.
# It does NOT:
#   - read TRAIN/TEST/RUL dataset files
#   - execute any historical cell
#   - construct regimes
#   - normalize FIT/DEV
#   - select a model feature set
#   - train a model
#   - select threshold/policy
#
# Purpose:
#   Recover the exact historical FD002 multi-operating-condition preprocessing
#   implementation before applying any normalization to FD004.

import hashlib
import json
from pathlib import Path


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630d", False)):
    raise RuntimeError(
        "STEP 6.30D must pass before STEP 6.30D-R1."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630D", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30D-R1."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630D", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30D-R1."
    )

if bool(
    globals().get(
        "FD004_HISTORICAL_PREPROCESSING_LOCATOR_COMPLETED_630DR1",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30D-R1 has already been completed."
    )


# =============================================================================
# 2) Notebook source scope
# =============================================================================

notebook_root_630dr1 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks"
)

if not notebook_root_630dr1.is_dir():
    raise RuntimeError(
        "Expected notebook directory was not found."
    )

notebook_paths_630dr1 = sorted(
    notebook_root_630dr1.rglob("*.ipynb")
)

if not notebook_paths_630dr1:
    raise RuntimeError(
        "No .ipynb files were found under the notebook directory."
    )


# =============================================================================
# 3) Source-only search terms
# =============================================================================

exact_priority_terms_630dr1 = [
    "FD002_REGIME_SEED_628C",
    "FD002_FIT_NORMALIZED_628C",
    "FD002_DEV_NORMALIZED_628C",
]

support_terms_630dr1 = [
    "regime_id",
    "KMeans",
    "kmeans",
    "StandardScaler",
    "standardscaler",
    "op_setting_1_z",
    "op_setting_2_z",
    "628C",
]


# =============================================================================
# 4) Parse notebook SOURCE only — never outputs
# =============================================================================

candidate_records_630dr1 = []

for notebook_path_630dr1 in notebook_paths_630dr1:

    try:
        notebook_obj_630dr1 = json.loads(
            notebook_path_630dr1.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc_630dr1:
        candidate_records_630dr1.append(
            {
                "notebook":
                    str(notebook_path_630dr1),

                "cell_index":
                    None,

                "cell_type":
                    "PARSE_ERROR",

                "priority_hits":
                    [],

                "support_hits":
                    [],

                "source_sha256":
                    None,

                "source":
                    "PARSE ERROR: "
                    + type(exc_630dr1).__name__,
            }
        )
        continue

    cells_630dr1 = notebook_obj_630dr1.get(
        "cells",
        []
    )

    for cell_index_630dr1, cell_630dr1 in enumerate(
        cells_630dr1
    ):

        source_raw_630dr1 = cell_630dr1.get(
            "source",
            []
        )

        if isinstance(source_raw_630dr1, list):
            source_630dr1 = "".join(
                str(x)
                for x in source_raw_630dr1
            )
        else:
            source_630dr1 = str(
                source_raw_630dr1
            )

        if not source_630dr1.strip():
            continue

        priority_hits_630dr1 = [
            term_630dr1
            for term_630dr1 in exact_priority_terms_630dr1
            if term_630dr1 in source_630dr1
        ]

        support_hits_630dr1 = [
            term_630dr1
            for term_630dr1 in support_terms_630dr1
            if term_630dr1 in source_630dr1
        ]

        keep_630dr1 = bool(
            priority_hits_630dr1
            or
            (
                "628C" in source_630dr1
                and
                any(
                    term_630dr1 in source_630dr1
                    for term_630dr1 in (
                        "regime",
                        "Regime",
                        "KMeans",
                        "kmeans",
                        "StandardScaler",
                        "standardscaler",
                        "_z",
                    )
                )
            )
        )

        if not keep_630dr1:
            continue

        source_sha_630dr1 = hashlib.sha256(
            source_630dr1.encode(
                "utf-8"
            )
        ).hexdigest()

        candidate_records_630dr1.append(
            {
                "notebook":
                    str(notebook_path_630dr1),

                "cell_index":
                    int(cell_index_630dr1),

                "cell_type":
                    str(
                        cell_630dr1.get(
                            "cell_type",
                            "unknown",
                        )
                    ),

                "priority_hits":
                    priority_hits_630dr1,

                "support_hits":
                    support_hits_630dr1,

                "source_sha256":
                    source_sha_630dr1,

                "source":
                    source_630dr1,
            }
        )


# =============================================================================
# 5) Rank candidates without executing them
# =============================================================================

def _candidate_rank_630dr1(record_630dr1):

    source_630dr1 = record_630dr1.get(
        "source",
        ""
    )

    score_630dr1 = 0

    score_630dr1 += 100 * len(
        record_630dr1.get(
            "priority_hits",
            [],
        )
    )

    for term_630dr1, weight_630dr1 in (
        ("FD002_REGIME_SEED_628C", 1000),
        ("FD002_FIT_NORMALIZED_628C", 800),
        ("FD002_DEV_NORMALIZED_628C", 800),
        ("regime_id", 100),
        ("KMeans", 100),
        ("kmeans", 100),
        ("StandardScaler", 75),
        ("standardscaler", 75),
        ("op_setting_1_z", 50),
        ("op_setting_2_z", 50),
    ):
        if term_630dr1 in source_630dr1:
            score_630dr1 += weight_630dr1

    return score_630dr1


candidate_records_630dr1 = sorted(
    candidate_records_630dr1,
    key=lambda r_630dr1: (
        -_candidate_rank_630dr1(
            r_630dr1
        ),
        r_630dr1.get(
            "notebook",
            "",
        ),
        (
            r_630dr1.get(
                "cell_index"
            )
            if r_630dr1.get(
                "cell_index"
            ) is not None
            else 10**12
        ),
    ),
)


# =============================================================================
# 6) Freeze locator evidence only
# =============================================================================

FD004_HISTORICAL_PREPROCESSING_SOURCE_CANDIDATES_630DR1 = [
    {
        "notebook":
            record_630dr1[
                "notebook"
            ],

        "cell_index":
            record_630dr1[
                "cell_index"
            ],

        "cell_type":
            record_630dr1[
                "cell_type"
            ],

        "priority_hits":
            list(
                record_630dr1[
                    "priority_hits"
                ]
            ),

        "support_hits":
            list(
                record_630dr1[
                    "support_hits"
                ]
            ),

        "source_sha256":
            record_630dr1[
                "source_sha256"
            ],
    }
    for record_630dr1 in candidate_records_630dr1
]


locator_payload_630dr1 = {
    "step":
        "6.30D-R1",

    "dataset":
        "FD004",

    "purpose":
        "recover historical FD002 multi-regime preprocessing source semantics",

    "notebook_root":
        str(
            notebook_root_630dr1
        ),

    "notebook_count":
        int(
            len(
                notebook_paths_630dr1
            )
        ),

    "candidate_count":
        int(
            len(
                candidate_records_630dr1
            )
        ),

    "candidates":
        FD004_HISTORICAL_PREPROCESSING_SOURCE_CANDIDATES_630DR1,

    "historical_cells_executed":
        False,

    "dataset_files_read":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "regime_assignment_performed":
        False,

    "normalization_performed":
        False,

    "feature_set_frozen":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,
}

FD004_HISTORICAL_PREPROCESSING_LOCATOR_SHA256_630DR1 = hashlib.sha256(
    json.dumps(
        locator_payload_630dr1,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
).hexdigest()

FD004_HISTORICAL_PREPROCESSING_LOCATOR_COMPLETED_630DR1 = True
all_checks_passed_630dr1 = True


# =============================================================================
# 7) Output
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30D-R1 — FD004 HISTORICAL MULTI-REGIME "
    "PREPROCESSING PROVENANCE LOCATOR"
)

print("=" * 174)

print("Notebook source files scanned    :", len(notebook_paths_630dr1))
print("Dataset files read               :", False)
print("Historical cells executed        :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)
print("Regime assignment performed      :", False)
print("Normalization performed          :", False)
print("Feature set frozen               :", False)
print("Model training performed         :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)

print("\nCandidate historical source cells:", len(candidate_records_630dr1))

if not candidate_records_630dr1:

    print(
        "\nNO MATCHING HISTORICAL 628C SOURCE CELL WAS FOUND."
    )

    print(
        "Stop here. Do not normalize FD004 until the historical "
        "multi-regime preprocessing semantics are resolved."
    )

else:

    max_show_630dr1 = min(
        8,
        len(
            candidate_records_630dr1
        ),
    )

    for rank_630dr1 in range(
        max_show_630dr1
    ):

        record_630dr1 = candidate_records_630dr1[
            rank_630dr1
        ]

        print("\n" + "-" * 120)

        print(
            "Candidate rank                  :",
            rank_630dr1 + 1,
        )

        print(
            "Notebook                        :",
            record_630dr1[
                "notebook"
            ],
        )

        print(
            "Cell index                      :",
            record_630dr1[
                "cell_index"
            ],
        )

        print(
            "Cell type                       :",
            record_630dr1[
                "cell_type"
            ],
        )

        print(
            "Priority hits                   :",
            record_630dr1[
                "priority_hits"
            ],
        )

        print(
            "Support hits                    :",
            record_630dr1[
                "support_hits"
            ],
        )

        print(
            "Source SHA-256                  :",
            record_630dr1[
                "source_sha256"
            ],
        )

        source_preview_630dr1 = record_630dr1[
            "source"
        ]

        if len(source_preview_630dr1) > 6000:
            source_preview_630dr1 = (
                source_preview_630dr1[
                    :6000
                ]
                + "\n...[SOURCE PREVIEW TRUNCATED]..."
            )

        print("\nSOURCE PREVIEW:\n")
        print(source_preview_630dr1)

print(
    "\nLocator evidence SHA-256         :",
    FD004_HISTORICAL_PREPROCESSING_LOCATOR_SHA256_630DR1,
)

print(
    "\nPASS — this was a source-only forensic locator. "
    "No dataset values or historical computation were executed."
)

print(
    "Next: inspect the recovered source evidence and freeze the exact "
    "FD004 multi-regime preprocessing protocol before transforming FIT/DEV."
)

In [ ]:
# STEP 6.30D-R2
# FD004 EXACT HISTORICAL 6.28C NORMALIZATION-SEMANTICS RECOVERY
#
# SOURCE-ONLY. Reads notebook JSON/source text only.
# Does NOT execute historical code or read TRAIN/TEST/RUL data.

import hashlib
import json
from pathlib import Path

if not bool(globals().get("all_checks_passed_630dr1", False)):
    raise RuntimeError("STEP 6.30D-R1 must pass before STEP 6.30D-R2.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630D", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30D-R2.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630D", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30D-R2.")

if bool(globals().get("FD004_HISTORICAL_628C_SOURCE_RECOVERED_630DR2", False)):
    raise RuntimeError("STEP 6.30D-R2 has already been completed.")

historical_notebook_630dr2 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks\07_fd002_replication.ipynb"
)
historical_cell_index_630dr2 = 2

expected_source_sha256_630dr2 = (
    "2e9c3bd0b9fa24d340b7fde6bbfbd6be68ef021d57298d72b124fdb341c7e998"
)

if not historical_notebook_630dr2.is_file():
    raise RuntimeError("Historical FD002 notebook was not found.")

notebook_obj_630dr2 = json.loads(
    historical_notebook_630dr2.read_text(encoding="utf-8")
)

cells_630dr2 = notebook_obj_630dr2.get("cells", [])

if historical_cell_index_630dr2 >= len(cells_630dr2):
    raise RuntimeError("Historical cell index 2 does not exist.")

cell_630dr2 = cells_630dr2[historical_cell_index_630dr2]

if cell_630dr2.get("cell_type") != "code":
    raise RuntimeError("Historical cell index 2 is not a code cell.")

source_raw_630dr2 = cell_630dr2.get("source", [])

if isinstance(source_raw_630dr2, list):
    source_630dr2 = "".join(str(x) for x in source_raw_630dr2)
else:
    source_630dr2 = str(source_raw_630dr2)

actual_source_sha256_630dr2 = hashlib.sha256(
    source_630dr2.encode("utf-8")
).hexdigest()

if actual_source_sha256_630dr2 != expected_source_sha256_630dr2:
    raise RuntimeError(
        "Historical STEP 6.28C source hash does not match the locator evidence."
    )

required_markers_630dr2 = [
    "STEP 6.28C",
    "FD002_REGIME_SCALER_628C",
    "FD002_REGIME_KMEANS_628C",
    "REGIME_REMAP_628C",
    "FD002_FIT_NORMALIZED_628C",
    "FD002_DEV_NORMALIZED_628C",
    "FD002_MODEL_FEATURES_628C",
]

missing_markers_630dr2 = [
    marker_630dr2
    for marker_630dr2 in required_markers_630dr2
    if marker_630dr2 not in source_630dr2
]

if missing_markers_630dr2:
    raise RuntimeError(
        "Expected historical 6.28C markers are missing: "
        + str(missing_markers_630dr2)
    )

source_lines_630dr2 = source_630dr2.splitlines()

focus_terms_630dr2 = [
    "condition-specific",
    "sensor normalization",
    "SENSOR",
    "sensor_",
    "std(",
    "ddof",
    "mean(",
    "StandardScaler",
    "MODEL_OP",
    "MODEL_FEATURES",
    "FIT_NORMALIZED",
    "DEV_NORMALIZED",
    "PREPROCESS",
    "CONTRACT",
]

matched_line_indices_630dr2 = []

for idx_630dr2, line_630dr2 in enumerate(source_lines_630dr2):
    if any(
        term_630dr2.lower() in line_630dr2.lower()
        for term_630dr2 in focus_terms_630dr2
    ):
        matched_line_indices_630dr2.append(idx_630dr2)

if not matched_line_indices_630dr2:
    raise RuntimeError(
        "No normalization-semantics lines were found in historical STEP 6.28C."
    )

selected_indices_630dr2 = set()

for idx_630dr2 in matched_line_indices_630dr2:
    start_630dr2 = max(0, idx_630dr2 - 4)
    stop_630dr2 = min(len(source_lines_630dr2), idx_630dr2 + 7)
    selected_indices_630dr2.update(
        range(start_630dr2, stop_630dr2)
    )

anchor_candidates_630dr2 = [
    idx_630dr2
    for idx_630dr2, line_630dr2 in enumerate(source_lines_630dr2)
    if (
        "condition-specific" in line_630dr2.lower()
        or
        "sensor normalization" in line_630dr2.lower()
    )
]

if anchor_candidates_630dr2:
    anchor_630dr2 = min(anchor_candidates_630dr2)

    selected_indices_630dr2.update(
        range(
            anchor_630dr2,
            len(source_lines_630dr2),
        )
    )

selected_indices_630dr2 = sorted(
    selected_indices_630dr2
)

excerpt_lines_630dr2 = [
    f"{idx_630dr2 + 1:04d}: {source_lines_630dr2[idx_630dr2]}"
    for idx_630dr2 in selected_indices_630dr2
]

FD004_HISTORICAL_628C_SOURCE_SHA256_630DR2 = (
    actual_source_sha256_630dr2
)

FD004_HISTORICAL_628C_NORMALIZATION_EXCERPT_630DR2 = (
    "\n".join(excerpt_lines_630dr2)
)

excerpt_sha256_630dr2 = hashlib.sha256(
    FD004_HISTORICAL_628C_NORMALIZATION_EXCERPT_630DR2.encode(
        "utf-8"
    )
).hexdigest()

FD004_HISTORICAL_628C_NORMALIZATION_EXCERPT_SHA256_630DR2 = (
    excerpt_sha256_630dr2
)

FD004_HISTORICAL_628C_SOURCE_RECOVERED_630DR2 = True
all_checks_passed_630dr2 = True

print("=" * 174)

print(
    "STEP 6.30D-R2 — FD004 EXACT HISTORICAL 6.28C "
    "NORMALIZATION-SEMANTICS RECOVERY"
)

print("=" * 174)

print(
    "Historical notebook              :",
    historical_notebook_630dr2,
)

print(
    "Historical cell index            :",
    historical_cell_index_630dr2,
)

print("Historical cell executed         :", False)
print("Dataset files read               :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)
print("Normalization performed          :", False)
print("Model training performed         :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)

print(
    "Historical source SHA-256        :",
    actual_source_sha256_630dr2,
)

print(
    "Expected source SHA-256 exact    :",
    True,
)

print(
    "Recovered source line count      :",
    len(source_lines_630dr2),
)

print(
    "Focused excerpt line count       :",
    len(excerpt_lines_630dr2),
)

print(
    "Focused excerpt SHA-256          :",
    excerpt_sha256_630dr2,
)

print(
    "\n--- EXACT HISTORICAL NORMALIZATION / "
    "FEATURE-CONTRACT SOURCE ---\n"
)

print(
    FD004_HISTORICAL_628C_NORMALIZATION_EXCERPT_630DR2
)

print(
    "\nPASS — exact historical STEP 6.28C normalization semantics "
    "were recovered from source only; no historical computation "
    "or dataset values were executed/read."
)

print(
    "Next: inspect this exact source, then prospectively freeze the "
    "FD004-specific regime/normalization/feature protocol before any "
    "FIT/DEV transformation."
)

In [ ]:
# STEP 6.30E
# FD004 PROSPECTIVE MULTI-REGIME PREPROCESSING + FEATURE-PROTOCOL FREEZE
#
# FREEZE ONLY.
# No regime assignment, normalization, model training, TEST/RUL access,
# threshold selection, or policy selection is performed here.

import hashlib
import json


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630d", False)):
    raise RuntimeError("STEP 6.30D must pass before STEP 6.30E.")

if not bool(globals().get("all_checks_passed_630dr1", False)):
    raise RuntimeError("STEP 6.30D-R1 must pass before STEP 6.30E.")

if not bool(globals().get("all_checks_passed_630dr2", False)):
    raise RuntimeError("STEP 6.30D-R2 must pass before STEP 6.30E.")

if not bool(
    globals().get(
        "FD004_HISTORICAL_628C_SOURCE_RECOVERED_630DR2",
        False,
    )
):
    raise RuntimeError(
        "Exact historical STEP 6.28C source recovery is not locked."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630D", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30E.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630D", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30E.")

if bool(globals().get("FD004_PREPROCESSING_PROTOCOL_LOCKED_630E", False)):
    raise RuntimeError("STEP 6.30E has already been completed.")


expected_historical_source_sha_630e = (
    "2e9c3bd0b9fa24d340b7fde6bbfbd6be68ef021d57298d72b124fdb341c7e998"
)

observed_historical_source_sha_630e = str(
    globals().get(
        "FD004_HISTORICAL_628C_SOURCE_SHA256_630DR2",
        "",
    )
)

if observed_historical_source_sha_630e != expected_historical_source_sha_630e:
    raise RuntimeError(
        "Recovered historical STEP 6.28C source fingerprint changed."
    )


# =============================================================================
# 2) FD004 dataset-specific feature contract
#
# STEP 6.30D established that all 24 raw candidate features are nonconstant
# on FD004 FIT. Therefore FD004 retains all of them.
#
# We transfer the historical CONDITION-AWARE NORMALIZATION ARCHITECTURE,
# not FD002's dataset-specific 17-feature list.
# =============================================================================

FD004_OPERATIONAL_CLUSTER_COLS_630E = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

FD004_MODEL_OP_COLS_630E = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]

FD004_SENSOR_COLS_630E = [
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21",
]

FD004_RAW_MODEL_FEATURES_630E = (
    FD004_MODEL_OP_COLS_630E
    +
    FD004_SENSOR_COLS_630E
)

FD004_MODEL_FEATURES_630E = [
    f"{feature_630e}_z"
    for feature_630e in FD004_RAW_MODEL_FEATURES_630E
]

if len(FD004_RAW_MODEL_FEATURES_630E) != 24:
    raise RuntimeError("FD004 raw model feature count must be exactly 24.")

if len(set(FD004_RAW_MODEL_FEATURES_630E)) != 24:
    raise RuntimeError("FD004 raw model features are not unique.")

if len(FD004_MODEL_FEATURES_630E) != 24:
    raise RuntimeError("FD004 normalized model feature count must be 24.")


# =============================================================================
# 3) Historical multi-regime preprocessing semantics transferred prospectively
# =============================================================================

FD004_N_REGIMES_630E = 6

# Historical implementation constant from exact STEP 6.28C source.
# This is transferred as a reproducibility setting, not a learned statistic.
FD004_REGIME_SEED_630E = 62803

FD004_KMEANS_N_INIT_630E = 20

FD004_REGIME_CENTER_CANONICALIZATION_630E = (
    "lexicographic_sort_of_inverse_transformed_cluster_centers_"
    "over_op_setting_1_op_setting_2_op_setting_3"
)

FD004_SENSOR_STD_DDOF_630E = 0
FD004_SENSOR_STD_EPS_630E = 1e-12

FD004_SENSOR_STD_FALLBACK_630E = (
    "if_regime_std_nonfinite_or_le_1e-12_use_FIT_global_sensor_std_ddof0;"
    "if_global_std_nonfinite_or_le_1e-12_use_1.0"
)

FD004_SENSOR_NORMALIZATION_630E = (
    "FIT_regime_specific_zscore"
)

FD004_MODEL_OP_NORMALIZATION_630E = (
    "FIT_global_StandardScaler"
)

FD004_DEV_ROLE_630E = "transform_only"

FD004_TEST_ROLE_630E = (
    "sealed_until_one_way_TEST_admission"
)


# =============================================================================
# 4) Freeze protocol contract
# =============================================================================

FD004_PREPROCESSING_PROTOCOL_630E = {
    "step": "6.30E",
    "dataset": "FD004",

    "freeze_only": True,
    "regime_assignment_performed": False,
    "normalization_performed": False,
    "windowing_performed": False,
    "model_training_performed": False,

    "feature_selection_source":
        "FD004_FIT_only_exact_nonconstant_audit_6.30D",

    "raw_candidate_feature_count": 24,
    "selected_raw_feature_count": 24,

    "selected_raw_features":
        list(FD004_RAW_MODEL_FEATURES_630E),

    "normalized_model_features":
        list(FD004_MODEL_FEATURES_630E),

    "historical_method_source":
        "FD002_STEP_6.28C",

    "historical_method_source_sha256":
        expected_historical_source_sha_630e,

    "operational_cluster_columns":
        list(FD004_OPERATIONAL_CLUSTER_COLS_630E),

    "n_regimes":
        int(FD004_N_REGIMES_630E),

    "regime_scaler":
        "StandardScaler_fit_on_FIT_only",

    "regime_method":
        "KMeans",

    "regime_seed":
        int(FD004_REGIME_SEED_630E),

    "kmeans_n_init":
        int(FD004_KMEANS_N_INIT_630E),

    "canonical_regime_id_rule":
        FD004_REGIME_CENTER_CANONICALIZATION_630E,

    "model_operating_columns":
        list(FD004_MODEL_OP_COLS_630E),

    "model_operating_normalization":
        FD004_MODEL_OP_NORMALIZATION_630E,

    "sensor_columns":
        list(FD004_SENSOR_COLS_630E),

    "sensor_normalization":
        FD004_SENSOR_NORMALIZATION_630E,

    "sensor_std_ddof":
        int(FD004_SENSOR_STD_DDOF_630E),

    "sensor_std_epsilon":
        float(FD004_SENSOR_STD_EPS_630E),

    "sensor_std_fallback":
        FD004_SENSOR_STD_FALLBACK_630E,

    "fit_role":
        "fit_all_preprocessing_parameters",

    "dev_role":
        FD004_DEV_ROLE_630E,

    "test_role":
        FD004_TEST_ROLE_630E,

    "test_contents_read": False,
    "official_rul_contents_read": False,

    "fd002_learned_statistics_inherited": False,
    "fd003_numerical_state_inherited": False,

    "threshold_selected": False,
    "policy_selected": False,
}


FD004_PREPROCESSING_PROTOCOL_SHA256_630E = hashlib.sha256(
    json.dumps(
        FD004_PREPROCESSING_PROTOCOL_630E,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


FD004_PREPROCESSING_PROTOCOL_LOCKED_630E = True

FD004_REGIME_ASSIGNMENT_PERFORMED_630E = False
FD004_NORMALIZATION_PERFORMED_630E = False
FD004_WINDOWING_PERFORMED_630E = False
FD004_MODEL_TRAINING_PERFORMED_630E = False
FD004_TEST_CONTENTS_READ_630E = False
FD004_RUL_CONTENTS_READ_630E = False
FD004_THRESHOLD_SELECTED_630E = False
FD004_POLICY_SELECTED_630E = False

all_checks_passed_630e = True


# =============================================================================
# 5) Report
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30E — FD004 PROSPECTIVE MULTI-REGIME "
    "PREPROCESSING + FEATURE-PROTOCOL FREEZE"
)

print("=" * 174)

print(
    "Protocol locked                 :",
    FD004_PREPROCESSING_PROTOCOL_LOCKED_630E,
)

print(
    "Regime assignment performed     :",
    FD004_REGIME_ASSIGNMENT_PERFORMED_630E,
)

print(
    "Normalization performed         :",
    FD004_NORMALIZATION_PERFORMED_630E,
)

print(
    "Windowing performed             :",
    FD004_WINDOWING_PERFORMED_630E,
)

print(
    "Model training performed        :",
    FD004_MODEL_TRAINING_PERFORMED_630E,
)

print(
    "TEST contents read              :",
    FD004_TEST_CONTENTS_READ_630E,
)

print(
    "Official RUL contents read      :",
    FD004_RUL_CONTENTS_READ_630E,
)

print(
    "Threshold selected              :",
    FD004_THRESHOLD_SELECTED_630E,
)

print(
    "Policy selected                 :",
    FD004_POLICY_SELECTED_630E,
)


print("\n--- FD004 DATASET-SPECIFIC FEATURE CONTRACT ---")

print(
    "Raw candidate features          :",
    24,
)

print(
    "Selected raw features           :",
    len(FD004_RAW_MODEL_FEATURES_630E),
)

print(
    "Model operating inputs          :",
    FD004_MODEL_OP_COLS_630E,
)

print(
    "Selected sensors                :",
    len(FD004_SENSOR_COLS_630E),
)

print(
    "Total normalized model features :",
    len(FD004_MODEL_FEATURES_630E),
)

print(
    "FD002 17-feature list inherited :",
    False,
)


print("\n--- FROZEN MULTI-REGIME PREPROCESSING METHOD ---")

print(
    "Regime inputs                   :",
    FD004_OPERATIONAL_CLUSTER_COLS_630E,
)

print(
    "Operating regimes               :",
    FD004_N_REGIMES_630E,
)

print(
    "KMeans seed / n_init            :",
    (
        FD004_REGIME_SEED_630E,
        FD004_KMEANS_N_INIT_630E,
    ),
)

print(
    "Canonical regime-ID rule        :",
    FD004_REGIME_CENTER_CANONICALIZATION_630E,
)

print(
    "Sensor normalization            :",
    FD004_SENSOR_NORMALIZATION_630E,
)

print(
    "Sensor STD ddof / epsilon       :",
    (
        FD004_SENSOR_STD_DDOF_630E,
        FD004_SENSOR_STD_EPS_630E,
    ),
)

print(
    "Sensor fallback rule            :",
    FD004_SENSOR_STD_FALLBACK_630E,
)

print(
    "Model-op normalization          :",
    FD004_MODEL_OP_NORMALIZATION_630E,
)

print(
    "DEV role                        :",
    FD004_DEV_ROLE_630E,
)

print(
    "\nHistorical STEP 6.28C SHA-256   :",
    expected_historical_source_sha_630e,
)

print(
    "FD004 preprocessing protocol SHA-256:",
    FD004_PREPROCESSING_PROTOCOL_SHA256_630E,
)

print(
    "\nPASS — the FD004 preprocessing architecture and 24-feature "
    "contract are now prospectively frozen before any regime fitting "
    "or normalization."
)

print(
    "Next: fit the operating-setting scaler and six-regime KMeans on "
    "FD004 FIT only, canonically assign FIT/DEV regime IDs, fit all "
    "sensor/model-op normalization parameters on FIT only, and transform "
    "FIT/DEV without touching TEST or official RUL."
)

In [ ]:
# STEP 6.30E-R1
# FD004 CROSS-DATASET FEATURE-CONTRACT PROVENANCE AUDIT
#
# SOURCE-ONLY FORENSIC STEP.
# No dataset values are read and no historical cell is executed.

import hashlib
import json
from pathlib import Path


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630e", False)):
    raise RuntimeError(
        "STEP 6.30E must pass before STEP 6.30E-R1."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630E", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30E-R1."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630E", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30E-R1."
    )

if bool(
    globals().get(
        "FD004_FEATURE_PROVENANCE_AUDIT_COMPLETED_630ER1",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30E-R1 has already been completed."
    )


# =============================================================================
# 2) Notebook source scope
# =============================================================================

notebook_root_630er1 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks"
)

if not notebook_root_630er1.is_dir():
    raise RuntimeError(
        "Expected notebook directory was not found."
    )

notebook_paths_630er1 = sorted(
    notebook_root_630er1.rglob("*.ipynb")
)

if not notebook_paths_630er1:
    raise RuntimeError(
        "No notebooks were found for the provenance audit."
    )


# =============================================================================
# 3) Cross-dataset feature-contract search terms
# =============================================================================

families_630er1 = {

    "FD001": [
        "FD001_MODEL_FEATURES",
        "FD001_SELECTED",
        "17 model features",
        "17-input feature",
        "sensor_2",
        "sensor_21",
    ],

    "FD002": [
        "Same 17-input feature structure retained from FD001",
        "FD002_MODEL_FEATURES_628C",
        "FD002_SENSOR_COLS_628C",
        "FD002_MODEL_OP_COLS_628C",
    ],

    "FD003": [
        "FD003_SELECTED_RAW_FEATURES_629E",
        "FD003_FIT_EXACT_CONSTANT",
        "exact constant",
        "nonconstant",
        "18 features",
        "sensor_10",
    ],
}


# =============================================================================
# 4) Parse notebook SOURCE only
# =============================================================================

records_630er1 = []

for notebook_path_630er1 in notebook_paths_630er1:

    try:

        notebook_obj_630er1 = json.loads(
            notebook_path_630er1.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue

    for cell_index_630er1, cell_630er1 in enumerate(
        notebook_obj_630er1.get(
            "cells",
            [],
        )
    ):

        if cell_630er1.get(
            "cell_type"
        ) != "code":
            continue

        source_raw_630er1 = cell_630er1.get(
            "source",
            [],
        )

        if isinstance(
            source_raw_630er1,
            list,
        ):
            source_630er1 = "".join(
                str(x)
                for x in source_raw_630er1
            )
        else:
            source_630er1 = str(
                source_raw_630er1
            )

        if not source_630er1.strip():
            continue

        # Exclude this forensic audit itself if notebook source
        # is already visible to the parser.
        if "STEP 6.30E-R1" in source_630er1:
            continue

        source_lower_630er1 = (
            source_630er1.lower()
        )

        family_hits_630er1 = {}
        total_score_630er1 = 0

        for (
            family_630er1,
            terms_630er1,
        ) in families_630er1.items():

            hits_630er1 = [
                term_630er1
                for term_630er1 in terms_630er1
                if term_630er1.lower()
                in source_lower_630er1
            ]

            if hits_630er1:

                family_hits_630er1[
                    family_630er1
                ] = hits_630er1

                total_score_630er1 += len(
                    hits_630er1
                )

        if not family_hits_630er1:
            continue


        # ---------------------------------------------------------------------
        # Produce focused feature-contract excerpt only.
        # ---------------------------------------------------------------------

        source_lines_630er1 = (
            source_630er1.splitlines()
        )

        interesting_indices_630er1 = []

        focus_terms_630er1 = [
            "feature",
            "sensor",
            "constant",
            "nonconstant",
            "selected",
            "model_op",
            "model_features",
            "17-input",
            "18 features",
            "retained from fd001",
        ]

        for (
            line_idx_630er1,
            line_630er1,
        ) in enumerate(
            source_lines_630er1
        ):

            line_lower_630er1 = (
                line_630er1.lower()
            )

            if any(
                term_630er1
                in line_lower_630er1
                for term_630er1
                in focus_terms_630er1
            ):
                interesting_indices_630er1.append(
                    line_idx_630er1
                )

        excerpt_indices_630er1 = set()

        for line_idx_630er1 in (
            interesting_indices_630er1
        ):

            excerpt_indices_630er1.update(
                range(
                    max(
                        0,
                        line_idx_630er1 - 2,
                    ),
                    min(
                        len(
                            source_lines_630er1
                        ),
                        line_idx_630er1 + 4,
                    ),
                )
            )

        excerpt_indices_630er1 = sorted(
            excerpt_indices_630er1
        )

        excerpt_630er1 = "\n".join(
            f"{idx_630er1 + 1:04d}: "
            f"{source_lines_630er1[idx_630er1]}"
            for idx_630er1
            in excerpt_indices_630er1
        )

        if len(excerpt_630er1) > 9000:

            excerpt_630er1 = (
                excerpt_630er1[:9000]
                +
                "\n...[EXCERPT TRUNCATED]..."
            )

        records_630er1.append(
            {
                "notebook":
                    str(
                        notebook_path_630er1
                    ),

                "cell_index":
                    int(
                        cell_index_630er1
                    ),

                "family_hits":
                    family_hits_630er1,

                "score":
                    int(
                        total_score_630er1
                    ),

                "source_sha256":
                    hashlib.sha256(
                        source_630er1.encode(
                            "utf-8"
                        )
                    ).hexdigest(),

                "excerpt":
                    excerpt_630er1,
            }
        )


# =============================================================================
# 5) Rank source candidates
# =============================================================================

records_630er1 = sorted(
    records_630er1,
    key=lambda r: (
        -r["score"],
        r["notebook"],
        r["cell_index"],
    ),
)

top_records_630er1 = (
    records_630er1[:12]
)


# =============================================================================
# 6) Freeze forensic evidence only
# =============================================================================

audit_payload_630er1 = {

    "step":
        "6.30E-R1",

    "dataset":
        "FD004",

    "purpose":
        (
            "resolve cross-dataset model-feature contract provenance "
            "before FD004 preprocessing execution"
        ),

    "notebook_count":
        len(
            notebook_paths_630er1
        ),

    "candidate_count":
        len(
            records_630er1
        ),

    "top_candidates":
        [
            {
                "notebook":
                    r["notebook"],

                "cell_index":
                    r["cell_index"],

                "family_hits":
                    r["family_hits"],

                "score":
                    r["score"],

                "source_sha256":
                    r["source_sha256"],
            }
            for r in top_records_630er1
        ],

    "dataset_files_read":
        False,

    "historical_cells_executed":
        False,

    "regime_assignment_performed":
        False,

    "normalization_performed":
        False,

    "feature_contract_modified":
        False,

    "model_training_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,
}


FD004_FEATURE_PROVENANCE_AUDIT_SHA256_630ER1 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630er1,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_FEATURE_PROVENANCE_CANDIDATES_630ER1 = (
    top_records_630er1
)

FD004_FEATURE_PROVENANCE_AUDIT_COMPLETED_630ER1 = True

FD004_FEATURE_CONTRACT_MODIFIED_630ER1 = False

all_checks_passed_630er1 = True


# =============================================================================
# 7) Output
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30E-R1 — FD004 CROSS-DATASET "
    "FEATURE-CONTRACT PROVENANCE AUDIT"
)

print("=" * 174)

print(
    "Dataset files read               :",
    False,
)

print(
    "Historical cells executed        :",
    False,
)

print(
    "Regime assignment performed      :",
    False,
)

print(
    "Normalization performed          :",
    False,
)

print(
    "Feature contract modified        :",
    False,
)

print(
    "Model training performed         :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "\nCandidate source cells            :",
    len(
        records_630er1
    ),
)


if not top_records_630er1:

    print(
        "\nNO FEATURE-CONTRACT PROVENANCE "
        "CANDIDATES FOUND."
    )

else:

    for (
        rank_630er1,
        record_630er1,
    ) in enumerate(
        top_records_630er1,
        start=1,
    ):

        print(
            "\n"
            +
            "-" * 120
        )

        print(
            "Candidate rank                   :",
            rank_630er1,
        )

        print(
            "Notebook                         :",
            record_630er1[
                "notebook"
            ],
        )

        print(
            "Cell index                       :",
            record_630er1[
                "cell_index"
            ],
        )

        print(
            "Family hits                      :",
            record_630er1[
                "family_hits"
            ],
        )

        print(
            "Score                            :",
            record_630er1[
                "score"
            ],
        )

        print(
            "Source SHA-256                   :",
            record_630er1[
                "source_sha256"
            ],
        )

        print(
            "\nSOURCE EXCERPT:\n"
        )

        print(
            record_630er1[
                "excerpt"
            ]
        )


print(
    "\nFeature-provenance audit SHA-256 :",
    FD004_FEATURE_PROVENANCE_AUDIT_SHA256_630ER1,
)

print(
    "\nPASS — this audit inspected notebook SOURCE ONLY. "
    "STEP 6.30E has not been used to fit or transform any FD004 data."
)

print(
    "Next: review the recovered FD001/FD002/FD003 feature-contract "
    "evidence and either reaffirm or prospectively supersede STEP 6.30E "
    "before any regime fitting or normalization."
)

In [ ]:
# STEP 6.30E-R2
# FD004 FEATURE-CONTRACT PROVENANCE RESOLUTION + 6.30E REAFFIRMATION
#
# RECORD/LOCK ONLY.
# No regime fitting, normalization, windowing, training, TEST/RUL access,
# threshold selection, or policy selection is performed here.

import hashlib
import json


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630e", False)):
    raise RuntimeError("STEP 6.30E must pass before STEP 6.30E-R2.")

if not bool(globals().get("all_checks_passed_630er1", False)):
    raise RuntimeError("STEP 6.30E-R1 must pass before STEP 6.30E-R2.")

if not bool(
    globals().get(
        "FD004_FEATURE_PROVENANCE_AUDIT_COMPLETED_630ER1",
        False,
    )
):
    raise RuntimeError("Feature-provenance audit is not locked.")

if bool(globals().get("FD004_REGIME_ASSIGNMENT_PERFORMED_630E", True)):
    raise RuntimeError("Regime fitting unexpectedly occurred before resolution.")

if bool(globals().get("FD004_NORMALIZATION_PERFORMED_630E", True)):
    raise RuntimeError("Normalization unexpectedly occurred before resolution.")

if bool(globals().get("FD004_MODEL_TRAINING_PERFORMED_630E", True)):
    raise RuntimeError("Model training unexpectedly occurred before resolution.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630E", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30E-R2.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630E", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30E-R2.")

if bool(globals().get("FD004_FEATURE_CONTRACT_RESOLVED_630ER2", False)):
    raise RuntimeError("STEP 6.30E-R2 has already been completed.")


# =============================================================================
# 2) Resolve the governing feature-selection rule
# =============================================================================

FD004_FEATURE_SELECTION_RULE_630ER2 = (
    "exclude FIT-exact-constant raw inputs only; "
    "retain all remaining nonconstant inputs in original raw-column order"
)

FD004_FEATURE_RULE_PROVENANCE_630ER2 = {
    "FD001":
        "explicit rule: retain FIT-nonconstant features; drop exact constants only",
    "FD002":
        "17-input structure was the retained historical feature outcome; "
        "it is not treated as a universal fixed dimension",
    "FD003":
        "explicitly reapplied FIT-exact-constant filtering and did not force "
        "the historical 17-feature signature",
    "FD004":
        "STEP 6.30D found zero exact-constant FIT raw candidate features, "
        "therefore all 24 raw candidates are retained",
}

expected_raw_features_630er2 = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21",
]

observed_raw_features_630er2 = [
    str(x)
    for x in globals().get("FD004_RAW_MODEL_FEATURES_630E", [])
]

if observed_raw_features_630er2 != expected_raw_features_630er2:
    raise RuntimeError(
        "STEP 6.30E feature list does not equal the resolved "
        "FD004 FIT-nonconstant feature outcome."
    )

observed_normalized_features_630er2 = [
    str(x)
    for x in globals().get("FD004_MODEL_FEATURES_630E", [])
]

expected_normalized_features_630er2 = [
    f"{x}_z"
    for x in expected_raw_features_630er2
]

if observed_normalized_features_630er2 != expected_normalized_features_630er2:
    raise RuntimeError(
        "STEP 6.30E normalized feature order is inconsistent "
        "with the resolved raw feature contract."
    )

if len(observed_raw_features_630er2) != 24:
    raise RuntimeError("Resolved FD004 feature count must be exactly 24.")


# =============================================================================
# 3) Reaffirm STEP 6.30E
# =============================================================================

FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2 = tuple(
    observed_raw_features_630er2
)

FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2 = tuple(
    observed_normalized_features_630er2
)

FD004_FEATURE_CONTRACT_RESOLUTION_630ER2 = {
    "step": "6.30E-R2",
    "dataset": "FD004",

    "resolution":
        "STEP 6.30E 24-feature contract reaffirmed",

    "governing_rule":
        FD004_FEATURE_SELECTION_RULE_630ER2,

    "raw_candidate_count": 24,
    "fit_exact_constant_count": 0,
    "selected_feature_count": 24,

    "selected_raw_features":
        list(FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2),

    "normalized_model_features":
        list(FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2),

    "fd001_fd002_fixed_17_dimension_inherited":
        False,

    "fd001_fd002_feature_selection_methodology_inherited":
        True,

    "fd003_rule_consistent":
        True,

    "step_630e_superseded":
        False,

    "step_630e_reaffirmed":
        True,

    "regime_assignment_performed":
        False,

    "normalization_performed":
        False,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "feature_provenance_audit_sha256":
        str(FD004_FEATURE_PROVENANCE_AUDIT_SHA256_630ER1),

    "preprocessing_protocol_sha256":
        str(FD004_PREPROCESSING_PROTOCOL_SHA256_630E),
}


FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2 = hashlib.sha256(
    json.dumps(
        FD004_FEATURE_CONTRACT_RESOLUTION_630ER2,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
).hexdigest()


FD004_FEATURE_CONTRACT_RESOLVED_630ER2 = True
FD004_PREPROCESSING_PROTOCOL_REAFFIRMED_630ER2 = True
FD004_PREPROCESSING_PROTOCOL_SUPERSEDED_630ER2 = False

FD004_REGIME_ASSIGNMENT_PERFORMED_630ER2 = False
FD004_NORMALIZATION_PERFORMED_630ER2 = False
FD004_WINDOWING_PERFORMED_630ER2 = False
FD004_MODEL_TRAINING_PERFORMED_630ER2 = False
FD004_TEST_CONTENTS_READ_630ER2 = False
FD004_RUL_CONTENTS_READ_630ER2 = False
FD004_THRESHOLD_SELECTED_630ER2 = False
FD004_POLICY_SELECTED_630ER2 = False

all_checks_passed_630er2 = True


# =============================================================================
# 4) Report
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30E-R2 — FD004 FEATURE-CONTRACT PROVENANCE "
    "RESOLUTION + 6.30E REAFFIRMATION"
)

print("=" * 174)

print(
    "Feature contract resolved        :",
    FD004_FEATURE_CONTRACT_RESOLVED_630ER2,
)

print(
    "STEP 6.30E reaffirmed            :",
    FD004_PREPROCESSING_PROTOCOL_REAFFIRMED_630ER2,
)

print(
    "STEP 6.30E superseded            :",
    FD004_PREPROCESSING_PROTOCOL_SUPERSEDED_630ER2,
)

print("Regime assignment performed      :", False)
print("Normalization performed          :", False)
print("Windowing performed              :", False)
print("Model training performed         :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)


print("\n--- RESOLVED GOVERNING RULE ---")

print(
    "Rule                             :",
    FD004_FEATURE_SELECTION_RULE_630ER2,
)

print("FIT exact-constant count         :", 0)

print(
    "Selected feature count           :",
    len(FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2),
)

print("Fixed historical 17 inherited   :", False)
print("Selection methodology inherited  :", True)


print("\n--- AUTHORITATIVE FD004 FEATURE CONTRACT ---")

print("Raw features:")

for idx_630er2, name_630er2 in enumerate(
    FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2,
    start=1,
):
    print(f"{idx_630er2:02d}. {name_630er2}")


print(
    "\nFeature-resolution SHA-256       :",
    FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2,
)

print(
    "\nPASS — the provenance ambiguity is resolved before any FD004 "
    "regime fitting or normalization. STEP 6.30E is reaffirmed, "
    "not superseded."
)

print(
    "Next: fit the frozen six-regime preprocessing system on FD004 FIT only, "
    "transform FIT/DEV only, and keep TEST and official RUL sealed."
)

In [ ]:
# STEP 6.30F-0
# FD004 PREPROCESSING INPUT-AUTHORITY LOCATOR
#
# METADATA / IDENTITY CHECK ONLY.
# No regime fitting, normalization, windowing, training, TEST/RUL access,
# threshold selection, or policy selection.

import pandas as pd

if not bool(globals().get("all_checks_passed_630er2", False)):
    raise RuntimeError("STEP 6.30E-R2 must pass before STEP 6.30F-0.")


candidate_names_630f0 = [
    "FD004_FIT_RAW_630C",
    "FD004_DEV_RAW_630C",
    "FD004_FIT_LABELED_RAW_630D",
    "FD004_DEV_LABELED_RAW_630D",
    "fit_raw_630d",
    "dev_raw_630d",

    "FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2",
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",

    "FD004_OPERATIONAL_CLUSTER_COLS_630E",
    "FD004_MODEL_OP_COLS_630E",
    "FD004_SENSOR_COLS_630E",

    "FD004_N_REGIMES_630E",
    "FD004_REGIME_SEED_630E",
    "FD004_SENSOR_STD_DDOF_630E",
    "FD004_SENSOR_STD_EPS_630E",
    "FD004_SENSOR_STD_FALLBACK_630E",

    "FD004_PREPROCESSING_PROTOCOL_SHA256_630E",
    "FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2",
]


rows_630f0 = []

for name_630f0 in candidate_names_630f0:

    present_630f0 = name_630f0 in globals()

    row_630f0 = {
        "name": name_630f0,
        "present": bool(present_630f0),
        "type": None,
        "shape_or_len": None,
        "columns_or_value": None,
    }

    if present_630f0:

        obj_630f0 = globals()[name_630f0]

        row_630f0["type"] = type(obj_630f0).__name__

        if isinstance(obj_630f0, pd.DataFrame):

            row_630f0["shape_or_len"] = tuple(
                obj_630f0.shape
            )

            row_630f0["columns_or_value"] = list(
                obj_630f0.columns
            )

        elif isinstance(
            obj_630f0,
            (list, tuple),
        ):

            row_630f0["shape_or_len"] = len(
                obj_630f0
            )

            row_630f0["columns_or_value"] = list(
                obj_630f0
            )

        elif isinstance(
            obj_630f0,
            (
                int,
                float,
                str,
                bool,
            ),
        ):

            row_630f0["columns_or_value"] = (
                obj_630f0
            )

        else:

            try:
                row_630f0["shape_or_len"] = tuple(
                    obj_630f0.shape
                )

            except Exception:

                try:
                    row_630f0["shape_or_len"] = len(
                        obj_630f0
                    )

                except Exception:
                    pass

    rows_630f0.append(
        row_630f0
    )


FD004_PREPROCESSING_INPUT_LOCATOR_630F0 = (
    pd.DataFrame(
        rows_630f0
    )
)


FD004_REGIME_ASSIGNMENT_PERFORMED_630F0 = False
FD004_NORMALIZATION_PERFORMED_630F0 = False
FD004_WINDOWING_PERFORMED_630F0 = False
FD004_MODEL_TRAINING_PERFORMED_630F0 = False
FD004_TEST_CONTENTS_READ_630F0 = False
FD004_RUL_CONTENTS_READ_630F0 = False
FD004_THRESHOLD_SELECTED_630F0 = False
FD004_POLICY_SELECTED_630F0 = False

all_checks_passed_630f0 = True


print("=" * 174)

print(
    "STEP 6.30F-0 — FD004 PREPROCESSING "
    "INPUT-AUTHORITY LOCATOR"
)

print("=" * 174)


print(
    "Regime assignment performed      :",
    False,
)

print(
    "Normalization performed          :",
    False,
)

print(
    "Windowing performed              :",
    False,
)

print(
    "Model training performed         :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)


print(
    "\n--- WHITELISTED PREPROCESSING INPUT OBJECTS ---"
)

print(
    FD004_PREPROCESSING_INPUT_LOCATOR_630F0
    .to_string(
        index=False
    )
)


print(
    "\nPASS — only whitelisted FD004 FIT/DEV "
    "preprocessing-authority objects were inspected. "
    "No preprocessing or model operation was executed."
)

In [ ]:
# STEP 6.30F
# FD004 FIT-ONLY SIX-REGIME PREPROCESSING EXECUTION + NORMALIZATION LOCK

import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630f(value):
    arr = np.ascontiguousarray(np.asarray(value))

    if arr.dtype.hasobject:
        raise RuntimeError(
            "Object-dtype array refused during STEP 6.30F hashing."
        )

    return hashlib.sha256(
        arr.tobytes(order="C")
    ).hexdigest()


def _sha_json_630f(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630f0", False)):
    raise RuntimeError(
        "STEP 6.30F-0 must pass before STEP 6.30F."
    )

if not bool(globals().get("all_checks_passed_630er2", False)):
    raise RuntimeError(
        "STEP 6.30E-R2 must pass before STEP 6.30F."
    )

required_630f = [
    "FD004_FIT_RAW_630C",
    "FD004_DEV_RAW_630C",
    "FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2",
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "FD004_OPERATIONAL_CLUSTER_COLS_630E",
    "FD004_MODEL_OP_COLS_630E",
    "FD004_SENSOR_COLS_630E",
    "FD004_N_REGIMES_630E",
    "FD004_REGIME_SEED_630E",
    "FD004_KMEANS_N_INIT_630E",
    "FD004_SENSOR_STD_DDOF_630E",
    "FD004_SENSOR_STD_EPS_630E",
    "FD004_PREPROCESSING_PROTOCOL_630E",
    "FD004_PREPROCESSING_PROTOCOL_SHA256_630E",
    "FD004_FEATURE_CONTRACT_RESOLUTION_630ER2",
    "FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2",
    "FD004_FIT_RAW_NUMERIC_SHA256_630C",
    "FD004_DEV_RAW_NUMERIC_SHA256_630C",
    "FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B",
]

missing_630f = [
    name
    for name in required_630f
    if name not in globals()
]

if missing_630f:
    raise RuntimeError(
        "Missing authoritative STEP 6.30F input object(s): "
        + ", ".join(missing_630f)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630F0", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30F."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630F0", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30F."
    )

if bool(globals().get("FD004_PREPROCESSING_EXECUTED_630F", False)):
    raise RuntimeError(
        "STEP 6.30F has already been completed."
    )


# =============================================================================
# 3) Re-certify frozen protocol + feature contract
# =============================================================================

if (
    _sha_json_630f(
        FD004_PREPROCESSING_PROTOCOL_630E
    )
    !=
    FD004_PREPROCESSING_PROTOCOL_SHA256_630E
):
    raise RuntimeError(
        "Frozen STEP 6.30E preprocessing protocol changed."
    )

if (
    _sha_json_630f(
        FD004_FEATURE_CONTRACT_RESOLUTION_630ER2
    )
    !=
    FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2
):
    raise RuntimeError(
        "Frozen STEP 6.30E-R2 feature resolution changed."
    )


raw_schema_630f = list(
    FD004_TRAIN_RAW_VERIFICATION_CONTRACT_630B[
        "raw_schema"
    ]
)

if (
    _sha_arr_630f(
        FD004_FIT_RAW_630C[
            raw_schema_630f
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
    !=
    FD004_FIT_RAW_NUMERIC_SHA256_630C
):
    raise RuntimeError(
        "Frozen FD004 FIT raw frame changed."
    )

if (
    _sha_arr_630f(
        FD004_DEV_RAW_630C[
            raw_schema_630f
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
    !=
    FD004_DEV_RAW_NUMERIC_SHA256_630C
):
    raise RuntimeError(
        "Frozen FD004 DEV raw frame changed."
    )


# =============================================================================
# 4) Resolve authoritative frozen configuration
# =============================================================================

raw_features_630f = list(
    FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2
)

model_features_630f = list(
    FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
)

cluster_cols_630f = list(
    FD004_OPERATIONAL_CLUSTER_COLS_630E
)

model_op_cols_630f = list(
    FD004_MODEL_OP_COLS_630E
)

sensor_cols_630f = list(
    FD004_SENSOR_COLS_630E
)


if len(raw_features_630f) != 24:
    raise RuntimeError(
        "FD004 authoritative raw feature count is not 24."
    )

if len(model_features_630f) != 24:
    raise RuntimeError(
        "FD004 authoritative normalized feature count is not 24."
    )

if raw_features_630f != model_op_cols_630f + sensor_cols_630f:
    raise RuntimeError(
        "FD004 authoritative feature order is inconsistent."
    )

if cluster_cols_630f != [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
]:
    raise RuntimeError(
        "Unexpected operating-regime input columns."
    )

if int(FD004_N_REGIMES_630E) != 6:
    raise RuntimeError(
        "FD004 regime count is not 6."
    )

if int(FD004_SENSOR_STD_DDOF_630E) != 0:
    raise RuntimeError(
        "FD004 sensor STD ddof is not 0."
    )

if float(FD004_SENSOR_STD_EPS_630E) != 1e-12:
    raise RuntimeError(
        "FD004 sensor STD epsilon is not exactly the frozen 1e-12."
    )


# =============================================================================
# 5) Work on copies only
# =============================================================================

fit_raw_630f = FD004_FIT_RAW_630C.copy(
    deep=True
)

dev_raw_630f = FD004_DEV_RAW_630C.copy(
    deep=True
)


# =============================================================================
# 6) FIT-only operating-setting regime scaler
# =============================================================================

fit_cluster_raw_630f = fit_raw_630f[
    cluster_cols_630f
].to_numpy(
    dtype=np.float64,
    copy=True,
)

dev_cluster_raw_630f = dev_raw_630f[
    cluster_cols_630f
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if not np.isfinite(
    fit_cluster_raw_630f
).all():
    raise RuntimeError(
        "Non-finite FIT operating-setting values detected."
    )

if not np.isfinite(
    dev_cluster_raw_630f
).all():
    raise RuntimeError(
        "Non-finite DEV operating-setting values detected."
    )


FD004_REGIME_SCALER_630F = StandardScaler()

fit_cluster_scaled_630f = (
    FD004_REGIME_SCALER_630F.fit_transform(
        fit_cluster_raw_630f
    )
)

dev_cluster_scaled_630f = (
    FD004_REGIME_SCALER_630F.transform(
        dev_cluster_raw_630f
    )
)


# =============================================================================
# 7) FIT-only six-regime KMeans
# =============================================================================

FD004_REGIME_KMEANS_630F = KMeans(
    n_clusters=int(
        FD004_N_REGIMES_630E
    ),
    random_state=int(
        FD004_REGIME_SEED_630E
    ),
    n_init=int(
        FD004_KMEANS_N_INIT_630E
    ),
)

fit_raw_cluster_labels_630f = (
    FD004_REGIME_KMEANS_630F
    .fit_predict(
        fit_cluster_scaled_630f
    )
    .astype(
        np.int64,
        copy=False,
    )
)

dev_raw_cluster_labels_630f = (
    FD004_REGIME_KMEANS_630F
    .predict(
        dev_cluster_scaled_630f
    )
    .astype(
        np.int64,
        copy=False,
    )
)


# =============================================================================
# 8) Canonical deterministic regime-ID remapping
# =============================================================================

cluster_centers_scaled_630f = np.asarray(
    FD004_REGIME_KMEANS_630F.cluster_centers_,
    dtype=np.float64,
)

cluster_centers_original_630f = (
    FD004_REGIME_SCALER_630F.inverse_transform(
        cluster_centers_scaled_630f
    )
)

canonical_old_labels_630f = np.asarray(
    sorted(
        range(
            int(
                FD004_N_REGIMES_630E
            )
        ),
        key=lambda old_label_630f: tuple(
            float(x)
            for x in cluster_centers_original_630f[
                old_label_630f
            ]
        ),
    ),
    dtype=np.int64,
)


FD004_REGIME_REMAP_630F = {
    int(old_label_630f):
        int(new_label_630f)
    for (
        new_label_630f,
        old_label_630f,
    ) in enumerate(
        canonical_old_labels_630f
    )
}


FD004_CANONICAL_REGIME_CENTERS_630F = np.asarray(
    [
        cluster_centers_original_630f[
            old_label_630f
        ]
        for old_label_630f
        in canonical_old_labels_630f
    ],
    dtype=np.float64,
)


FD004_FIT_REGIME_IDS_630F = np.asarray(
    [
        FD004_REGIME_REMAP_630F[
            int(label_630f)
        ]
        for label_630f
        in fit_raw_cluster_labels_630f
    ],
    dtype=np.int64,
)


FD004_DEV_REGIME_IDS_630F = np.asarray(
    [
        FD004_REGIME_REMAP_630F[
            int(label_630f)
        ]
        for label_630f
        in dev_raw_cluster_labels_630f
    ],
    dtype=np.int64,
)


expected_regime_ids_630f = np.arange(
    int(
        FD004_N_REGIMES_630E
    ),
    dtype=np.int64,
)

fit_present_regimes_630f = np.unique(
    FD004_FIT_REGIME_IDS_630F
)

dev_present_regimes_630f = np.unique(
    FD004_DEV_REGIME_IDS_630F
)

if not np.array_equal(
    fit_present_regimes_630f,
    expected_regime_ids_630f,
):
    raise RuntimeError(
        "Not all six canonical regimes are represented in FD004 FIT."
    )

if not np.array_equal(
    dev_present_regimes_630f,
    expected_regime_ids_630f,
):
    raise RuntimeError(
        "Not all six canonical regimes are represented in FD004 DEV."
    )


fit_regime_counts_630f = np.bincount(
    FD004_FIT_REGIME_IDS_630F,
    minlength=int(
        FD004_N_REGIMES_630E
    ),
).astype(
    np.int64
)

dev_regime_counts_630f = np.bincount(
    FD004_DEV_REGIME_IDS_630F,
    minlength=int(
        FD004_N_REGIMES_630E
    ),
).astype(
    np.int64
)


# =============================================================================
# 9) FIT-only sensor normalization statistics
# =============================================================================

sensor_count_630f = len(
    sensor_cols_630f
)

regime_count_630f = int(
    FD004_N_REGIMES_630E
)


FD004_FIT_GLOBAL_SENSOR_MEANS_630F = np.zeros(
    sensor_count_630f,
    dtype=np.float64,
)

FD004_FIT_GLOBAL_SENSOR_STDS_630F = np.zeros(
    sensor_count_630f,
    dtype=np.float64,
)

FD004_FIT_REGIME_SENSOR_MEANS_630F = np.zeros(
    (
        regime_count_630f,
        sensor_count_630f,
    ),
    dtype=np.float64,
)

FD004_FIT_REGIME_SENSOR_STDS_RAW_630F = np.zeros(
    (
        regime_count_630f,
        sensor_count_630f,
    ),
    dtype=np.float64,
)

FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F = np.zeros(
    (
        regime_count_630f,
        sensor_count_630f,
    ),
    dtype=np.float64,
)

FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F = np.zeros(
    (
        regime_count_630f,
        sensor_count_630f,
    ),
    dtype=bool,
)


fit_sensor_raw_630f = fit_raw_630f[
    sensor_cols_630f
].to_numpy(
    dtype=np.float64,
    copy=True,
)

dev_sensor_raw_630f = dev_raw_630f[
    sensor_cols_630f
].to_numpy(
    dtype=np.float64,
    copy=True,
)


if not np.isfinite(
    fit_sensor_raw_630f
).all():
    raise RuntimeError(
        "Non-finite FIT sensor values detected."
    )

if not np.isfinite(
    dev_sensor_raw_630f
).all():
    raise RuntimeError(
        "Non-finite DEV sensor values detected."
    )


# FIT-global fallback statistics

for sensor_index_630f in range(
    sensor_count_630f
):

    sensor_fit_all_630f = (
        fit_sensor_raw_630f[
            :,
            sensor_index_630f
        ]
    )

    global_mean_630f = float(
        np.mean(
            sensor_fit_all_630f
        )
    )

    global_std_630f = float(
        np.std(
            sensor_fit_all_630f,
            ddof=int(
                FD004_SENSOR_STD_DDOF_630E
            ),
        )
    )

    if (
        not np.isfinite(
            global_std_630f
        )
        or
        global_std_630f
        <= float(
            FD004_SENSOR_STD_EPS_630E
        )
    ):
        global_std_630f = 1.0

    FD004_FIT_GLOBAL_SENSOR_MEANS_630F[
        sensor_index_630f
    ] = global_mean_630f

    FD004_FIT_GLOBAL_SENSOR_STDS_630F[
        sensor_index_630f
    ] = global_std_630f


# Regime-specific FIT statistics

for regime_id_630f in range(
    regime_count_630f
):

    regime_mask_630f = (
        FD004_FIT_REGIME_IDS_630F
        ==
        regime_id_630f
    )

    if not np.any(
        regime_mask_630f
    ):
        raise RuntimeError(
            f"FD004 FIT canonical regime "
            f"{regime_id_630f} is empty."
        )

    regime_sensor_values_630f = (
        fit_sensor_raw_630f[
            regime_mask_630f
        ]
    )

    regime_means_630f = np.mean(
        regime_sensor_values_630f,
        axis=0,
        dtype=np.float64,
    )

    regime_stds_raw_630f = np.std(
        regime_sensor_values_630f,
        axis=0,
        ddof=int(
            FD004_SENSOR_STD_DDOF_630E
        ),
        dtype=np.float64,
    )

    effective_stds_630f = (
        regime_stds_raw_630f.copy()
    )

    fallback_mask_630f = (
        ~np.isfinite(
            effective_stds_630f
        )
    ) | (
        effective_stds_630f
        <= float(
            FD004_SENSOR_STD_EPS_630E
        )
    )

    effective_stds_630f[
        fallback_mask_630f
    ] = FD004_FIT_GLOBAL_SENSOR_STDS_630F[
        fallback_mask_630f
    ]

    second_fallback_mask_630f = (
        ~np.isfinite(
            effective_stds_630f
        )
    ) | (
        effective_stds_630f
        <= float(
            FD004_SENSOR_STD_EPS_630E
        )
    )

    effective_stds_630f[
        second_fallback_mask_630f
    ] = 1.0

    if not np.isfinite(
        regime_means_630f
    ).all():
        raise RuntimeError(
            f"Non-finite FIT regime sensor means "
            f"for regime {regime_id_630f}."
        )

    if not np.isfinite(
        effective_stds_630f
    ).all():
        raise RuntimeError(
            f"Non-finite effective FIT regime sensor STDs "
            f"for regime {regime_id_630f}."
        )

    if np.any(
        effective_stds_630f
        <= float(
            FD004_SENSOR_STD_EPS_630E
        )
    ):
        raise RuntimeError(
            f"Invalid effective sensor STD remained "
            f"in regime {regime_id_630f}."
        )

    FD004_FIT_REGIME_SENSOR_MEANS_630F[
        regime_id_630f
    ] = regime_means_630f

    FD004_FIT_REGIME_SENSOR_STDS_RAW_630F[
        regime_id_630f
    ] = regime_stds_raw_630f

    FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F[
        regime_id_630f
    ] = effective_stds_630f

    FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F[
        regime_id_630f
    ] = (
        fallback_mask_630f
        |
        second_fallback_mask_630f
    )


# =============================================================================
# 10) Apply regime-specific sensor normalization
# =============================================================================

fit_sensor_z_630f = np.empty_like(
    fit_sensor_raw_630f,
    dtype=np.float64,
)

dev_sensor_z_630f = np.empty_like(
    dev_sensor_raw_630f,
    dtype=np.float64,
)


for regime_id_630f in range(
    regime_count_630f
):

    fit_mask_630f = (
        FD004_FIT_REGIME_IDS_630F
        ==
        regime_id_630f
    )

    dev_mask_630f = (
        FD004_DEV_REGIME_IDS_630F
        ==
        regime_id_630f
    )

    regime_means_630f = (
        FD004_FIT_REGIME_SENSOR_MEANS_630F[
            regime_id_630f
        ]
    )

    regime_stds_630f = (
        FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F[
            regime_id_630f
        ]
    )

    fit_sensor_z_630f[
        fit_mask_630f
    ] = (
        fit_sensor_raw_630f[
            fit_mask_630f
        ]
        -
        regime_means_630f
    ) / regime_stds_630f

    dev_sensor_z_630f[
        dev_mask_630f
    ] = (
        dev_sensor_raw_630f[
            dev_mask_630f
        ]
        -
        regime_means_630f
    ) / regime_stds_630f


# =============================================================================
# 11) FIT-global operating-setting normalization
# =============================================================================

FD004_MODEL_OP_SCALER_630F = StandardScaler()

fit_model_op_z_630f = (
    FD004_MODEL_OP_SCALER_630F
    .fit_transform(
        fit_raw_630f[
            model_op_cols_630f
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
)

dev_model_op_z_630f = (
    FD004_MODEL_OP_SCALER_630F
    .transform(
        dev_raw_630f[
            model_op_cols_630f
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )
)


# =============================================================================
# 12) Construct final 24-feature normalized matrices
# =============================================================================

fit_model_matrix_630f = np.concatenate(
    [
        fit_model_op_z_630f,
        fit_sensor_z_630f,
    ],
    axis=1,
)

dev_model_matrix_630f = np.concatenate(
    [
        dev_model_op_z_630f,
        dev_sensor_z_630f,
    ],
    axis=1,
)


if fit_model_matrix_630f.shape != (
    FD004_FIT_RAW_630C.shape[0],
    24,
):
    raise RuntimeError(
        "Unexpected normalized FD004 FIT matrix shape."
    )

if dev_model_matrix_630f.shape != (
    FD004_DEV_RAW_630C.shape[0],
    24,
):
    raise RuntimeError(
        "Unexpected normalized FD004 DEV matrix shape."
    )

if not np.isfinite(
    fit_model_matrix_630f
).all():
    raise RuntimeError(
        "Normalized FD004 FIT matrix contains non-finite values."
    )

if not np.isfinite(
    dev_model_matrix_630f
).all():
    raise RuntimeError(
        "Normalized FD004 DEV matrix contains non-finite values."
    )


# =============================================================================
# 13) Construct identity-preserving normalized frames
# =============================================================================

FD004_FIT_NORMALIZED_630F = pd.DataFrame(
    fit_model_matrix_630f,
    columns=model_features_630f,
    index=fit_raw_630f.index,
)

FD004_DEV_NORMALIZED_630F = pd.DataFrame(
    dev_model_matrix_630f,
    columns=model_features_630f,
    index=dev_raw_630f.index,
)


FD004_FIT_NORMALIZED_630F.insert(
    0,
    "regime_id",
    FD004_FIT_REGIME_IDS_630F,
)

FD004_FIT_NORMALIZED_630F.insert(
    0,
    "cycle",
    fit_raw_630f[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    ),
)

FD004_FIT_NORMALIZED_630F.insert(
    0,
    "unit_id",
    fit_raw_630f[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


FD004_DEV_NORMALIZED_630F.insert(
    0,
    "regime_id",
    FD004_DEV_REGIME_IDS_630F,
)

FD004_DEV_NORMALIZED_630F.insert(
    0,
    "cycle",
    dev_raw_630f[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    ),
)

FD004_DEV_NORMALIZED_630F.insert(
    0,
    "unit_id",
    dev_raw_630f[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    ),
)


fit_identity_exact_630f = bool(
    np.array_equal(
        FD004_FIT_NORMALIZED_630F[
            ["unit_id", "cycle"]
        ].to_numpy(
            dtype=np.int64
        ),
        fit_raw_630f[
            ["unit_id", "cycle"]
        ].to_numpy(
            dtype=np.int64
        ),
    )
)

dev_identity_exact_630f = bool(
    np.array_equal(
        FD004_DEV_NORMALIZED_630F[
            ["unit_id", "cycle"]
        ].to_numpy(
            dtype=np.int64
        ),
        dev_raw_630f[
            ["unit_id", "cycle"]
        ].to_numpy(
            dtype=np.int64
        ),
    )
)


if not fit_identity_exact_630f:
    raise RuntimeError(
        "FIT row identity/order changed during preprocessing."
    )

if not dev_identity_exact_630f:
    raise RuntimeError(
        "DEV row identity/order changed during preprocessing."
    )


# =============================================================================
# 14) Evidence fingerprints
# =============================================================================

FD004_REGIME_SCALER_MEAN_SHA256_630F = _sha_arr_630f(
    FD004_REGIME_SCALER_630F.mean_
)

FD004_REGIME_SCALER_SCALE_SHA256_630F = _sha_arr_630f(
    FD004_REGIME_SCALER_630F.scale_
)

FD004_KMEANS_CENTERS_SCALED_SHA256_630F = _sha_arr_630f(
    cluster_centers_scaled_630f
)

FD004_CANONICAL_REGIME_CENTERS_SHA256_630F = _sha_arr_630f(
    FD004_CANONICAL_REGIME_CENTERS_630F
)

FD004_FIT_REGIME_IDS_SHA256_630F = _sha_arr_630f(
    FD004_FIT_REGIME_IDS_630F
)

FD004_DEV_REGIME_IDS_SHA256_630F = _sha_arr_630f(
    FD004_DEV_REGIME_IDS_630F
)

FD004_SENSOR_REGIME_MEANS_SHA256_630F = _sha_arr_630f(
    FD004_FIT_REGIME_SENSOR_MEANS_630F
)

FD004_SENSOR_REGIME_STDS_EFFECTIVE_SHA256_630F = _sha_arr_630f(
    FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F
)

FD004_SENSOR_FALLBACK_MASK_SHA256_630F = _sha_arr_630f(
    FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F.astype(
        np.uint8
    )
)

FD004_MODEL_OP_SCALER_MEAN_SHA256_630F = _sha_arr_630f(
    FD004_MODEL_OP_SCALER_630F.mean_
)

FD004_MODEL_OP_SCALER_SCALE_SHA256_630F = _sha_arr_630f(
    FD004_MODEL_OP_SCALER_630F.scale_
)

FD004_FIT_MODEL_MATRIX_SHA256_630F = _sha_arr_630f(
    fit_model_matrix_630f
)

FD004_DEV_MODEL_MATRIX_SHA256_630F = _sha_arr_630f(
    dev_model_matrix_630f
)


# =============================================================================
# 15) Freeze execution contract
# =============================================================================

FD004_PREPROCESSING_EXECUTION_CONTRACT_630F = {

    "step":
        "6.30F",

    "dataset":
        "FD004",

    "preprocessing_protocol_sha256":
        FD004_PREPROCESSING_PROTOCOL_SHA256_630E,

    "feature_resolution_sha256":
        FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2,

    "fit_only_parameter_estimation":
        True,

    "dev_transform_only":
        True,

    "operational_cluster_columns":
        cluster_cols_630f,

    "n_regimes":
        regime_count_630f,

    "regime_seed":
        int(
            FD004_REGIME_SEED_630E
        ),

    "kmeans_n_init":
        int(
            FD004_KMEANS_N_INIT_630E
        ),

    "canonical_regime_rule":
        FD004_REGIME_CENTER_CANONICALIZATION_630E,

    "fit_regime_counts":
        [
            int(x)
            for x in fit_regime_counts_630f
        ],

    "dev_regime_counts":
        [
            int(x)
            for x in dev_regime_counts_630f
        ],

    "sensor_std_ddof":
        int(
            FD004_SENSOR_STD_DDOF_630E
        ),

    "sensor_std_epsilon":
        float(
            FD004_SENSOR_STD_EPS_630E
        ),

    "sensor_fallback_count":
        int(
            np.count_nonzero(
                FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F
            )
        ),

    "model_operating_normalization":
        "FIT_global_StandardScaler",

    "sensor_normalization":
        "FIT_regime_specific_zscore",

    "model_feature_count":
        24,

    "fit_normalized_shape":
        [
            int(x)
            for x in fit_model_matrix_630f.shape
        ],

    "dev_normalized_shape":
        [
            int(x)
            for x in dev_model_matrix_630f.shape
        ],

    "fit_row_identity_exact":
        fit_identity_exact_630f,

    "dev_row_identity_exact":
        dev_identity_exact_630f,

    "regime_scaler_mean_sha256":
        FD004_REGIME_SCALER_MEAN_SHA256_630F,

    "regime_scaler_scale_sha256":
        FD004_REGIME_SCALER_SCALE_SHA256_630F,

    "kmeans_centers_scaled_sha256":
        FD004_KMEANS_CENTERS_SCALED_SHA256_630F,

    "canonical_regime_centers_sha256":
        FD004_CANONICAL_REGIME_CENTERS_SHA256_630F,

    "fit_regime_ids_sha256":
        FD004_FIT_REGIME_IDS_SHA256_630F,

    "dev_regime_ids_sha256":
        FD004_DEV_REGIME_IDS_SHA256_630F,

    "sensor_regime_means_sha256":
        FD004_SENSOR_REGIME_MEANS_SHA256_630F,

    "sensor_regime_stds_effective_sha256":
        FD004_SENSOR_REGIME_STDS_EFFECTIVE_SHA256_630F,

    "sensor_fallback_mask_sha256":
        FD004_SENSOR_FALLBACK_MASK_SHA256_630F,

    "model_op_scaler_mean_sha256":
        FD004_MODEL_OP_SCALER_MEAN_SHA256_630F,

    "model_op_scaler_scale_sha256":
        FD004_MODEL_OP_SCALER_SCALE_SHA256_630F,

    "fit_model_matrix_sha256":
        FD004_FIT_MODEL_MATRIX_SHA256_630F,

    "dev_model_matrix_sha256":
        FD004_DEV_MODEL_MATRIX_SHA256_630F,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "windowing_performed":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,
}


FD004_PREPROCESSING_EXECUTION_CONTRACT_SHA256_630F = (
    _sha_json_630f(
        FD004_PREPROCESSING_EXECUTION_CONTRACT_630F
    )
)


# =============================================================================
# 16) Lock arrays/state
# =============================================================================

FD004_FIT_REGIME_IDS_630F.setflags(
    write=False
)

FD004_DEV_REGIME_IDS_630F.setflags(
    write=False
)

FD004_CANONICAL_REGIME_CENTERS_630F.setflags(
    write=False
)

FD004_FIT_GLOBAL_SENSOR_MEANS_630F.setflags(
    write=False
)

FD004_FIT_GLOBAL_SENSOR_STDS_630F.setflags(
    write=False
)

FD004_FIT_REGIME_SENSOR_MEANS_630F.setflags(
    write=False
)

FD004_FIT_REGIME_SENSOR_STDS_RAW_630F.setflags(
    write=False
)

FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F.setflags(
    write=False
)

FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F.setflags(
    write=False
)


FD004_REGIME_ASSIGNMENT_PERFORMED_630F = True
FD004_NORMALIZATION_PERFORMED_630F = True

FD004_PREPROCESSING_EXECUTED_630F = True
FD004_PREPROCESSING_LOCKED_630F = True

FD004_WINDOWING_PERFORMED_630F = False
FD004_MODEL_TRAINING_PERFORMED_630F = False

FD004_TEST_CONTENTS_READ_630F = False
FD004_RUL_CONTENTS_READ_630F = False

FD004_THRESHOLD_SELECTED_630F = False
FD004_POLICY_SELECTED_630F = False

all_checks_passed_630f = True


# =============================================================================
# 17) Report
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30F — FD004 FIT-ONLY SIX-REGIME "
    "PREPROCESSING EXECUTION + NORMALIZATION LOCK"
)

print("=" * 174)

print("Regime fitting source            : FIT only")
print("DEV used to fit preprocessing   :", False)

print(
    "Regime assignment performed      :",
    True,
)

print(
    "Normalization performed          :",
    True,
)

print(
    "Windowing performed              :",
    False,
)

print(
    "Model training performed         :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)


print("\n--- FROZEN EPSILON CHECK ---")

print(
    "Sensor STD epsilon repr          :",
    repr(
        float(
            FD004_SENSOR_STD_EPS_630E
        )
    ),
)

print(
    "Sensor STD epsilon exact 1e-12   :",
    float(
        FD004_SENSOR_STD_EPS_630E
    )
    ==
    1e-12,
)


print("\n--- SIX-REGIME FIT ---")

print(
    "FIT regime counts               :",
    fit_regime_counts_630f.tolist(),
)

print(
    "DEV regime counts               :",
    dev_regime_counts_630f.tolist(),
)

print(
    "All 6 regimes in FIT / DEV      :",
    (
        len(
            fit_present_regimes_630f
        ) == 6,
        len(
            dev_present_regimes_630f
        ) == 6,
    ),
)


print(
    "\nCanonical regime centers in original operating-setting units:"
)

print(
    pd.DataFrame(
        FD004_CANONICAL_REGIME_CENTERS_630F,
        columns=cluster_cols_630f,
        index=[
            f"regime_{i}"
            for i in range(
                regime_count_630f
            )
        ],
    ).to_string()
)


print("\n--- NORMALIZATION ---")

print(
    "Normalized FIT model shape      :",
    fit_model_matrix_630f.shape,
)

print(
    "Normalized DEV model shape      :",
    dev_model_matrix_630f.shape,
)

print(
    "Model feature count             :",
    fit_model_matrix_630f.shape[1],
)

print(
    "FIT / DEV finite                :",
    (
        bool(
            np.isfinite(
                fit_model_matrix_630f
            ).all()
        ),
        bool(
            np.isfinite(
                dev_model_matrix_630f
            ).all()
        ),
    ),
)

print(
    "FIT / DEV row identity exact    :",
    (
        fit_identity_exact_630f,
        dev_identity_exact_630f,
    ),
)

print(
    "Sensor STD fallback uses        :",
    int(
        np.count_nonzero(
            FD004_FIT_REGIME_SENSOR_STD_FALLBACK_USED_630F
        )
    ),
)


print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Canonical centers SHA-256       :",
    FD004_CANONICAL_REGIME_CENTERS_SHA256_630F,
)

print(
    "FIT regime IDs SHA-256          :",
    FD004_FIT_REGIME_IDS_SHA256_630F,
)

print(
    "DEV regime IDs SHA-256          :",
    FD004_DEV_REGIME_IDS_SHA256_630F,
)

print(
    "Sensor means SHA-256            :",
    FD004_SENSOR_REGIME_MEANS_SHA256_630F,
)

print(
    "Sensor effective STDs SHA-256   :",
    FD004_SENSOR_REGIME_STDS_EFFECTIVE_SHA256_630F,
)

print(
    "FIT normalized matrix SHA-256   :",
    FD004_FIT_MODEL_MATRIX_SHA256_630F,
)

print(
    "DEV normalized matrix SHA-256   :",
    FD004_DEV_MODEL_MATRIX_SHA256_630F,
)

print(
    "Preprocessing contract SHA-256  :",
    FD004_PREPROCESSING_EXECUTION_CONTRACT_SHA256_630F,
)


print(
    "\nPASS — the frozen six-regime preprocessing system was fit using "
    "FD004 FIT only and applied to FIT/DEV with the locked 24-feature "
    "contract."
)

print(
    "TEST and official RUL remain sealed. Next: construct leakage-safe "
    "30-cycle FIT/DEV windows from these locked normalized matrices and "
    "align the already-frozen TRAIN-derived RUL targets."
)

In [ ]:
# STEP 6.30G
# FD004 LEAKAGE-SAFE 30-CYCLE FIT/DEV WINDOW CONSTRUCTION + TARGET ALIGNMENT LOCK

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630g(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30G hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_630g(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630f", False)):
    raise RuntimeError("STEP 6.30F must pass before STEP 6.30G.")

required_630g = [
    "FD004_FIT_NORMALIZED_630F",
    "FD004_DEV_NORMALIZED_630F",
    "FD004_FIT_MODEL_MATRIX_SHA256_630F",
    "FD004_DEV_MODEL_MATRIX_SHA256_630F",
    "FD004_PREPROCESSING_EXECUTION_CONTRACT_630F",
    "FD004_PREPROCESSING_EXECUTION_CONTRACT_SHA256_630F",
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "FD004_FIT_RUL_630D",
    "FD004_DEV_RUL_630D",
    "FD004_FIT_RUL_SHA256_630D",
    "FD004_DEV_RUL_SHA256_630D",
    "FD004_FIT_ENGINE_IDS_630C",
    "FD004_DEV_ENGINE_IDS_630C",
]

missing_630g = [name for name in required_630g if name not in globals()]

if missing_630g:
    raise RuntimeError(
        "Missing authoritative STEP 6.30G input object(s): "
        + ", ".join(missing_630g)
    )

if not bool(globals().get("FD004_PREPROCESSING_LOCKED_630F", False)):
    raise RuntimeError("FD004 preprocessing is not locked.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630F", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30G.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630F", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30G.")

if bool(globals().get("FD004_WINDOWING_PERFORMED_630G", False)):
    raise RuntimeError("STEP 6.30G has already been completed.")


# =============================================================================
# 3) Re-certify frozen preprocessing + targets
# =============================================================================

if (
    _sha_json_630g(FD004_PREPROCESSING_EXECUTION_CONTRACT_630F)
    != FD004_PREPROCESSING_EXECUTION_CONTRACT_SHA256_630F
):
    raise RuntimeError("STEP 6.30F preprocessing contract changed.")

model_features_630g = list(
    FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
)

if len(model_features_630g) != 24:
    raise RuntimeError("FD004 normalized model feature count is not 24.")

fit_matrix_630g = FD004_FIT_NORMALIZED_630F[
    model_features_630g
].to_numpy(
    dtype=np.float64,
    copy=True,
)

dev_matrix_630g = FD004_DEV_NORMALIZED_630F[
    model_features_630g
].to_numpy(
    dtype=np.float64,
    copy=True,
)

if (
    _sha_arr_630g(fit_matrix_630g)
    != FD004_FIT_MODEL_MATRIX_SHA256_630F
):
    raise RuntimeError("Frozen normalized FD004 FIT matrix changed.")

if (
    _sha_arr_630g(dev_matrix_630g)
    != FD004_DEV_MODEL_MATRIX_SHA256_630F
):
    raise RuntimeError("Frozen normalized FD004 DEV matrix changed.")

if (
    _sha_arr_630g(FD004_FIT_RUL_630D)
    != FD004_FIT_RUL_SHA256_630D
):
    raise RuntimeError("Frozen FD004 FIT RUL vector changed.")

if (
    _sha_arr_630g(FD004_DEV_RUL_630D)
    != FD004_DEV_RUL_SHA256_630D
):
    raise RuntimeError("Frozen FD004 DEV RUL vector changed.")


# =============================================================================
# 4) Freeze windowing semantics
# =============================================================================

FD004_WINDOW_LENGTH_630G = 30
FD004_WINDOW_STRIDE_630G = 1

FD004_WINDOW_TARGET_RULE_630G = (
    "RUL at final cycle of each 30-cycle window"
)


# =============================================================================
# 5) Canonical per-partition window constructor
# =============================================================================

def _build_windows_630g(
    normalized_df,
    row_rul,
    frozen_engine_ids,
    partition_name,
):

    feature_matrix = normalized_df[
        model_features_630g
    ].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    unit_ids = normalized_df[
        "unit_id"
    ].to_numpy(
        dtype=np.int64,
        copy=True,
    )

    cycles = normalized_df[
        "cycle"
    ].to_numpy(
        dtype=np.int64,
        copy=True,
    )

    row_rul = np.asarray(
        row_rul,
        dtype=np.int64,
    ).reshape(-1)

    frozen_engine_ids = np.asarray(
        frozen_engine_ids,
        dtype=np.int64,
    ).reshape(-1)

    if feature_matrix.shape[0] != row_rul.size:
        raise RuntimeError(
            f"{partition_name} normalized rows and RUL length disagree."
        )

    if (
        feature_matrix.shape[0] != unit_ids.size
        or
        unit_ids.size != cycles.size
    ):
        raise RuntimeError(
            f"{partition_name} normalized identity vectors are misaligned."
        )

    observed_ids = np.unique(unit_ids)

    if not np.array_equal(
        observed_ids,
        frozen_engine_ids,
    ):
        raise RuntimeError(
            f"{partition_name} engine membership differs from frozen split."
        )

    x_parts = []
    y_parts = []
    unit_parts = []
    endpoint_cycle_parts = []
    terminal_cycle_parts = []

    expected_window_count = 0

    for engine_id in frozen_engine_ids:

        mask = unit_ids == int(engine_id)

        engine_x = feature_matrix[mask]
        engine_cycles = cycles[mask]
        engine_rul = row_rul[mask]

        if engine_x.shape[0] < FD004_WINDOW_LENGTH_630G:
            raise RuntimeError(
                f"{partition_name} engine {engine_id} has fewer than 30 rows."
            )

        if not np.array_equal(
            engine_cycles,
            np.arange(
                int(engine_cycles[0]),
                int(engine_cycles[-1]) + 1,
                dtype=np.int64,
            ),
        ):
            raise RuntimeError(
                f"{partition_name} engine {engine_id} cycles are not contiguous."
            )

        engine_terminal_cycle = int(
            engine_cycles[-1]
        )

        expected_rul = (
            engine_terminal_cycle
            -
            engine_cycles
        ).astype(
            np.int64
        )

        if not np.array_equal(
            engine_rul,
            expected_rul,
        ):
            raise RuntimeError(
                f"{partition_name} engine {engine_id} RUL alignment changed."
            )

        n_engine_windows = (
            engine_x.shape[0]
            -
            FD004_WINDOW_LENGTH_630G
            +
            1
        )

        expected_window_count += n_engine_windows

        for start_idx in range(
            0,
            n_engine_windows,
            FD004_WINDOW_STRIDE_630G,
        ):

            end_idx = (
                start_idx
                +
                FD004_WINDOW_LENGTH_630G
            )

            endpoint_idx = end_idx - 1

            x_parts.append(
                engine_x[
                    start_idx:end_idx
                ]
            )

            y_parts.append(
                np.float32(
                    engine_rul[
                        endpoint_idx
                    ]
                )
            )

            unit_parts.append(
                np.int64(
                    engine_id
                )
            )

            endpoint_cycle_parts.append(
                np.int64(
                    engine_cycles[
                        endpoint_idx
                    ]
                )
            )

            terminal_cycle_parts.append(
                np.int64(
                    engine_terminal_cycle
                )
            )

    X = np.stack(
        x_parts,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    y = np.asarray(
        y_parts,
        dtype=np.float32,
    )

    window_unit_ids = np.asarray(
        unit_parts,
        dtype=np.int64,
    )

    endpoint_cycles = np.asarray(
        endpoint_cycle_parts,
        dtype=np.int64,
    )

    terminal_cycles = np.asarray(
        terminal_cycle_parts,
        dtype=np.int64,
    )

    if X.shape != (
        expected_window_count,
        FD004_WINDOW_LENGTH_630G,
        len(model_features_630g),
    ):
        raise RuntimeError(
            f"{partition_name} window tensor shape mismatch."
        )

    if y.shape != (
        expected_window_count,
    ):
        raise RuntimeError(
            f"{partition_name} target vector shape mismatch."
        )

    if not np.array_equal(
        y.astype(np.int64),
        terminal_cycles
        -
        endpoint_cycles,
    ):
        raise RuntimeError(
            f"{partition_name} endpoint-target alignment failed."
        )

    if not np.isfinite(X).all():
        raise RuntimeError(
            f"{partition_name} window tensor contains non-finite values."
        )

    if not np.isfinite(y).all():
        raise RuntimeError(
            f"{partition_name} target vector contains non-finite values."
        )

    return {
        "X":
            X,

        "y":
            y,

        "window_unit_ids":
            window_unit_ids,

        "endpoint_cycles":
            endpoint_cycles,

        "terminal_cycles":
            terminal_cycles,

        "expected_window_count":
            int(
                expected_window_count
            ),
    }


# =============================================================================
# 6) Construct FIT + DEV windows
# =============================================================================

fit_windows_630g = _build_windows_630g(
    FD004_FIT_NORMALIZED_630F,
    FD004_FIT_RUL_630D,
    FD004_FIT_ENGINE_IDS_630C,
    "FIT",
)

dev_windows_630g = _build_windows_630g(
    FD004_DEV_NORMALIZED_630F,
    FD004_DEV_RUL_630D,
    FD004_DEV_ENGINE_IDS_630C,
    "DEV",
)


X_FIT_FD004_630G = fit_windows_630g["X"]

y_FIT_FD004_630G = fit_windows_630g["y"]

FIT_WINDOW_UNIT_IDS_FD004_630G = fit_windows_630g[
    "window_unit_ids"
]

FIT_WINDOW_ENDPOINT_CYCLES_FD004_630G = fit_windows_630g[
    "endpoint_cycles"
]

FIT_WINDOW_TERMINAL_CYCLES_FD004_630G = fit_windows_630g[
    "terminal_cycles"
]


X_DEV_FD004_630G = dev_windows_630g["X"]

y_DEV_FD004_630G = dev_windows_630g["y"]

DEV_WINDOW_UNIT_IDS_FD004_630G = dev_windows_630g[
    "window_unit_ids"
]

DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G = dev_windows_630g[
    "endpoint_cycles"
]

DEV_WINDOW_TERMINAL_CYCLES_FD004_630G = dev_windows_630g[
    "terminal_cycles"
]


# =============================================================================
# 7) Analytical count checks
# =============================================================================

expected_fit_windows_630g = int(
    FD004_FIT_NORMALIZED_630F.shape[0]
    -
    len(
        FD004_FIT_ENGINE_IDS_630C
    )
    *
    (
        FD004_WINDOW_LENGTH_630G
        -
        1
    )
)

expected_dev_windows_630g = int(
    FD004_DEV_NORMALIZED_630F.shape[0]
    -
    len(
        FD004_DEV_ENGINE_IDS_630C
    )
    *
    (
        FD004_WINDOW_LENGTH_630G
        -
        1
    )
)

if (
    expected_fit_windows_630g
    !=
    fit_windows_630g[
        "expected_window_count"
    ]
):
    raise RuntimeError(
        "Analytical FIT window count disagrees with construction."
    )

if (
    expected_dev_windows_630g
    !=
    dev_windows_630g[
        "expected_window_count"
    ]
):
    raise RuntimeError(
        "Analytical DEV window count disagrees with construction."
    )


# =============================================================================
# 8) Membership checks
# =============================================================================

fit_engine_membership_exact_630g = bool(
    np.array_equal(
        np.unique(
            FIT_WINDOW_UNIT_IDS_FD004_630G
        ),
        np.asarray(
            FD004_FIT_ENGINE_IDS_630C,
            dtype=np.int64,
        ),
    )
)

dev_engine_membership_exact_630g = bool(
    np.array_equal(
        np.unique(
            DEV_WINDOW_UNIT_IDS_FD004_630G
        ),
        np.asarray(
            FD004_DEV_ENGINE_IDS_630C,
            dtype=np.int64,
        ),
    )
)

if not fit_engine_membership_exact_630g:
    raise RuntimeError(
        "FIT window engine membership is not exact."
    )

if not dev_engine_membership_exact_630g:
    raise RuntimeError(
        "DEV window engine membership is not exact."
    )


# =============================================================================
# 9) Evidence fingerprints
# =============================================================================

FD004_X_FIT_SHA256_630G = _sha_arr_630g(
    X_FIT_FD004_630G
)

FD004_Y_FIT_SHA256_630G = _sha_arr_630g(
    y_FIT_FD004_630G
)

FD004_FIT_WINDOW_UNIT_IDS_SHA256_630G = _sha_arr_630g(
    FIT_WINDOW_UNIT_IDS_FD004_630G
)

FD004_FIT_WINDOW_ENDPOINT_CYCLES_SHA256_630G = _sha_arr_630g(
    FIT_WINDOW_ENDPOINT_CYCLES_FD004_630G
)

FD004_FIT_WINDOW_TERMINAL_CYCLES_SHA256_630G = _sha_arr_630g(
    FIT_WINDOW_TERMINAL_CYCLES_FD004_630G
)


FD004_X_DEV_SHA256_630G = _sha_arr_630g(
    X_DEV_FD004_630G
)

FD004_Y_DEV_SHA256_630G = _sha_arr_630g(
    y_DEV_FD004_630G
)

FD004_DEV_WINDOW_UNIT_IDS_SHA256_630G = _sha_arr_630g(
    DEV_WINDOW_UNIT_IDS_FD004_630G
)

FD004_DEV_WINDOW_ENDPOINT_CYCLES_SHA256_630G = _sha_arr_630g(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G
)

FD004_DEV_WINDOW_TERMINAL_CYCLES_SHA256_630G = _sha_arr_630g(
    DEV_WINDOW_TERMINAL_CYCLES_FD004_630G
)


# =============================================================================
# 10) Freeze window contract
# =============================================================================

FD004_WINDOW_CONTRACT_630G = {

    "step":
        "6.30G",

    "dataset":
        "FD004",

    "preprocessing_contract_sha256":
        FD004_PREPROCESSING_EXECUTION_CONTRACT_SHA256_630F,

    "window_length":
        30,

    "stride":
        1,

    "feature_count":
        24,

    "feature_order":
        list(
            model_features_630g
        ),

    "target_rule":
        FD004_WINDOW_TARGET_RULE_630G,

    "fit_window_shape":
        [
            int(x)
            for x in X_FIT_FD004_630G.shape
        ],

    "dev_window_shape":
        [
            int(x)
            for x in X_DEV_FD004_630G.shape
        ],

    "fit_target_shape":
        [
            int(x)
            for x in y_FIT_FD004_630G.shape
        ],

    "dev_target_shape":
        [
            int(x)
            for x in y_DEV_FD004_630G.shape
        ],

    "fit_engine_membership_exact":
        fit_engine_membership_exact_630g,

    "dev_engine_membership_exact":
        dev_engine_membership_exact_630g,

    "fit_target_min":
        float(
            y_FIT_FD004_630G.min()
        ),

    "fit_target_max":
        float(
            y_FIT_FD004_630G.max()
        ),

    "dev_target_min":
        float(
            y_DEV_FD004_630G.min()
        ),

    "dev_target_max":
        float(
            y_DEV_FD004_630G.max()
        ),

    "x_fit_sha256":
        FD004_X_FIT_SHA256_630G,

    "y_fit_sha256":
        FD004_Y_FIT_SHA256_630G,

    "x_dev_sha256":
        FD004_X_DEV_SHA256_630G,

    "y_dev_sha256":
        FD004_Y_DEV_SHA256_630G,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "model_training_performed":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,
}


FD004_WINDOW_CONTRACT_SHA256_630G = _sha_json_630g(
    FD004_WINDOW_CONTRACT_630G
)


# =============================================================================
# 11) Lock arrays/state
# =============================================================================

for arr_630g in [
    X_FIT_FD004_630G,
    y_FIT_FD004_630G,
    FIT_WINDOW_UNIT_IDS_FD004_630G,
    FIT_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    FIT_WINDOW_TERMINAL_CYCLES_FD004_630G,
    X_DEV_FD004_630G,
    y_DEV_FD004_630G,
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    DEV_WINDOW_TERMINAL_CYCLES_FD004_630G,
]:
    arr_630g.setflags(
        write=False
    )


FD004_WINDOWING_PERFORMED_630G = True

FD004_WINDOWING_LOCKED_630G = True

FD004_MODEL_TRAINING_PERFORMED_630G = False

FD004_TEST_CONTENTS_READ_630G = False

FD004_RUL_CONTENTS_READ_630G = False

FD004_THRESHOLD_SELECTED_630G = False

FD004_POLICY_SELECTED_630G = False

all_checks_passed_630g = True


# =============================================================================
# 12) Report
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30G — FD004 LEAKAGE-SAFE 30-CYCLE "
    "FIT/DEV WINDOW CONSTRUCTION + TARGET ALIGNMENT LOCK"
)

print("=" * 174)

print("Window length / stride           :", (30, 1))
print("Model feature count              :", 24)
print("Target                           : endpoint uncapped TRAIN-derived RUL")
print("Model training performed         :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)


print("\n--- FIT WINDOWS ---")

print(
    "X_FIT shape / dtype              :",
    (
        X_FIT_FD004_630G.shape,
        X_FIT_FD004_630G.dtype,
    ),
)

print(
    "y_FIT shape / dtype              :",
    (
        y_FIT_FD004_630G.shape,
        y_FIT_FD004_630G.dtype,
    ),
)

print(
    "Expected FIT windows             :",
    expected_fit_windows_630g,
)

print(
    "FIT engine membership exact      :",
    fit_engine_membership_exact_630g,
)

print(
    "FIT target min / max             :",
    (
        float(
            y_FIT_FD004_630G.min()
        ),
        float(
            y_FIT_FD004_630G.max()
        ),
    ),
)


print("\n--- DEV WINDOWS ---")

print(
    "X_DEV shape / dtype              :",
    (
        X_DEV_FD004_630G.shape,
        X_DEV_FD004_630G.dtype,
    ),
)

print(
    "y_DEV shape / dtype              :",
    (
        y_DEV_FD004_630G.shape,
        y_DEV_FD004_630G.dtype,
    ),
)

print(
    "Expected DEV windows             :",
    expected_dev_windows_630g,
)

print(
    "DEV engine membership exact      :",
    dev_engine_membership_exact_630g,
)

print(
    "DEV target min / max             :",
    (
        float(
            y_DEV_FD004_630G.min()
        ),
        float(
            y_DEV_FD004_630G.max()
        ),
    ),
)


print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "X_FIT SHA-256                    :",
    FD004_X_FIT_SHA256_630G,
)

print(
    "y_FIT SHA-256                    :",
    FD004_Y_FIT_SHA256_630G,
)

print(
    "X_DEV SHA-256                    :",
    FD004_X_DEV_SHA256_630G,
)

print(
    "y_DEV SHA-256                    :",
    FD004_Y_DEV_SHA256_630G,
)

print(
    "FIT window-unit IDs SHA-256      :",
    FD004_FIT_WINDOW_UNIT_IDS_SHA256_630G,
)

print(
    "DEV window-unit IDs SHA-256      :",
    FD004_DEV_WINDOW_UNIT_IDS_SHA256_630G,
)

print(
    "Window contract SHA-256          :",
    FD004_WINDOW_CONTRACT_SHA256_630G,
)


print(
    "\nPASS — leakage-safe 30-cycle FD004 FIT/DEV windows are now "
    "constructed from the locked normalized features, with uncapped "
    "TRAIN-derived RUL aligned at each window endpoint."
)

print(
    "TEST and official RUL remain sealed. Next: recover/verify the "
    "cross-dataset GRU architecture-transfer rule and prospectively "
    "freeze the FD004 architecture/training protocol before any model training."
)

In [ ]:
# STEP 6.30H-0
# FD004 CROSS-DATASET GRU ARCHITECTURE + TRAINING-PROTOCOL SOURCE LOCATOR
#
# SOURCE-ONLY FORENSIC STEP.
# No historical cells are executed and no dataset/model operation is performed.

import hashlib
import json
from pathlib import Path


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630g", False)):
    raise RuntimeError("STEP 6.30G must pass before STEP 6.30H-0.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630G", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30H-0.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630G", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30H-0.")

if bool(globals().get("FD004_ARCH_TRAIN_SOURCE_LOCATOR_COMPLETED_630H0", False)):
    raise RuntimeError("STEP 6.30H-0 has already been completed.")


# =============================================================================
# 2) Notebook source scope
# =============================================================================

notebook_root_630h0 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks"
)

if not notebook_root_630h0.is_dir():
    raise RuntimeError("Expected notebook directory was not found.")

notebook_paths_630h0 = sorted(
    notebook_root_630h0.rglob("*.ipynb")
)

if not notebook_paths_630h0:
    raise RuntimeError("No .ipynb files were found under the notebook directory.")


# =============================================================================
# 3) Exact provenance markers
# =============================================================================

priority_terms_630h0 = [
    "FD002GRURegressor628EB",
    "STEP 6.29G",
    "STEP 6.29H",
    "FD003GRURegressor629H",
    "FD003_ARCHITECTURE_CONTRACT",
    "STEP 6.29I",
    "FD003_TRAINING_PROTOCOL",
]

support_terms_630h0 = [
    "hidden_size",
    "num_layers",
    "dense",
    "dropout",
    "nn.GRU",
    "MSELoss",
    "Adam",
    "early",
    "patience",
    "DEV_MAE",
    "batch_size",
    "max_epochs",
    "restore",
]


# =============================================================================
# 4) Parse notebook SOURCE only
# =============================================================================

records_630h0 = []

for notebook_path_630h0 in notebook_paths_630h0:

    try:
        notebook_obj_630h0 = json.loads(
            notebook_path_630h0.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        continue

    cells_630h0 = notebook_obj_630h0.get("cells", [])

    for cell_index_630h0, cell_630h0 in enumerate(cells_630h0):

        if cell_630h0.get("cell_type") != "code":
            continue

        source_raw_630h0 = cell_630h0.get("source", [])

        if isinstance(source_raw_630h0, list):
            source_630h0 = "".join(
                str(x)
                for x in source_raw_630h0
            )
        else:
            source_630h0 = str(source_raw_630h0)

        if not source_630h0.strip():
            continue

        # Exclude this locator itself if already saved into the notebook.
        if "STEP 6.30H-0" in source_630h0:
            continue

        priority_hits_630h0 = [
            term
            for term in priority_terms_630h0
            if term in source_630h0
        ]

        support_hits_630h0 = [
            term
            for term in support_terms_630h0
            if term.lower() in source_630h0.lower()
        ]

        keep_630h0 = bool(
            priority_hits_630h0
            or (
                (
                    "6.29H" in source_630h0
                    or "6.29I" in source_630h0
                    or "628EB" in source_630h0
                )
                and support_hits_630h0
            )
        )

        if not keep_630h0:
            continue

        source_lines_630h0 = source_630h0.splitlines()

        interesting_indices_630h0 = []

        focus_terms_630h0 = [
            "class ",
            "GRU",
            "hidden",
            "layer",
            "dense",
            "dropout",
            "fc1",
            "fc2",
            "forward",
            "MSELoss",
            "Adam",
            "lr",
            "batch",
            "epoch",
            "early",
            "patience",
            "DEV_MAE",
            "restore",
            "scheduler",
            "clipping",
            "seed",
            "contract",
        ]

        for line_idx_630h0, line_630h0 in enumerate(source_lines_630h0):
            line_lower_630h0 = line_630h0.lower()

            if any(
                term.lower() in line_lower_630h0
                for term in focus_terms_630h0
            ):
                interesting_indices_630h0.append(
                    line_idx_630h0
                )

        excerpt_indices_630h0 = set()

        for line_idx_630h0 in interesting_indices_630h0:
            excerpt_indices_630h0.update(
                range(
                    max(0, line_idx_630h0 - 2),
                    min(
                        len(source_lines_630h0),
                        line_idx_630h0 + 4,
                    ),
                )
            )

        excerpt_indices_630h0 = sorted(
            excerpt_indices_630h0
        )

        excerpt_630h0 = "\n".join(
            f"{idx + 1:04d}: {source_lines_630h0[idx]}"
            for idx in excerpt_indices_630h0
        )

        if len(excerpt_630h0) > 12000:
            excerpt_630h0 = (
                excerpt_630h0[:12000]
                + "\n...[EXCERPT TRUNCATED]..."
            )

        score_630h0 = (
            100 * len(priority_hits_630h0)
            +
            len(support_hits_630h0)
        )

        records_630h0.append(
            {
                "notebook": str(notebook_path_630h0),
                "cell_index": int(cell_index_630h0),
                "priority_hits": priority_hits_630h0,
                "support_hits": support_hits_630h0,
                "score": int(score_630h0),
                "source_sha256": hashlib.sha256(
                    source_630h0.encode("utf-8")
                ).hexdigest(),
                "excerpt": excerpt_630h0,
            }
        )


# =============================================================================
# 5) Rank and retain top evidence
# =============================================================================

records_630h0 = sorted(
    records_630h0,
    key=lambda r: (
        -r["score"],
        r["notebook"],
        r["cell_index"],
    ),
)

top_records_630h0 = records_630h0[:10]

if not top_records_630h0:
    raise RuntimeError(
        "No historical GRU architecture/training source candidates were found."
    )


# =============================================================================
# 6) Freeze locator evidence only
# =============================================================================

FD004_ARCH_TRAIN_SOURCE_CANDIDATES_630H0 = top_records_630h0

locator_payload_630h0 = {
    "step": "6.30H-0",
    "dataset": "FD004",
    "purpose":
        "recover exact prior GRU architecture-transfer and training-protocol source",
    "notebook_count": int(len(notebook_paths_630h0)),
    "candidate_count": int(len(records_630h0)),
    "top_candidates": [
        {
            "notebook": r["notebook"],
            "cell_index": r["cell_index"],
            "priority_hits": list(r["priority_hits"]),
            "support_hits": list(r["support_hits"]),
            "score": int(r["score"]),
            "source_sha256": r["source_sha256"],
        }
        for r in top_records_630h0
    ],
    "historical_cells_executed": False,
    "dataset_files_read": False,
    "model_instantiated": False,
    "model_training_performed": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
    "threshold_selected": False,
    "policy_selected": False,
}

FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0 = hashlib.sha256(
    json.dumps(
        locator_payload_630h0,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
).hexdigest()

FD004_ARCH_TRAIN_SOURCE_LOCATOR_COMPLETED_630H0 = True
all_checks_passed_630h0 = True


# =============================================================================
# 7) Report
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30H-0 — FD004 CROSS-DATASET GRU ARCHITECTURE "
    "+ TRAINING-PROTOCOL SOURCE LOCATOR"
)

print("=" * 174)

print("Historical cells executed        :", False)
print("Dataset files read               :", False)
print("Model instantiated               :", False)
print("Model training performed         :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)

print("\nCandidate source cells            :", len(records_630h0))

for rank_630h0, record_630h0 in enumerate(
    top_records_630h0,
    start=1,
):

    print("\n" + "-" * 120)

    print(
        "Candidate rank                   :",
        rank_630h0,
    )

    print(
        "Notebook                         :",
        record_630h0["notebook"],
    )

    print(
        "Cell index                       :",
        record_630h0["cell_index"],
    )

    print(
        "Priority hits                    :",
        record_630h0["priority_hits"],
    )

    print(
        "Support hits                     :",
        record_630h0["support_hits"],
    )

    print(
        "Score                            :",
        record_630h0["score"],
    )

    print(
        "Source SHA-256                   :",
        record_630h0["source_sha256"],
    )

    print("\nSOURCE EXCERPT:\n")
    print(record_630h0["excerpt"])

print(
    "\nLocator evidence SHA-256         :",
    FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0,
)

print(
    "\nPASS — only notebook SOURCE was inspected. "
    "No model was instantiated or trained, and TEST/RUL remain sealed."
)

print(
    "Next: inspect the recovered architecture/training evidence, then "
    "prospectively freeze the FD004 GRU architecture and single-run "
    "training protocol before initialization."
)

In [ ]:
# STEP 6.30H
# FD004 GRU ARCHITECTURE + SINGLE-RUN TRAINING-PROTOCOL PROSPECTIVE FREEZE
#
# FREEZE ONLY:
#   - no persistent model instance
#   - no optimizer
#   - no forward pass
#   - no training
#   - no DEV scoring
#   - no TEST / official RUL access

import hashlib
import json
import dis
import torch
import torch.nn as nn


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630g", False)):
    raise RuntimeError("STEP 6.30G must pass before STEP 6.30H.")

if not bool(globals().get("all_checks_passed_630h0", False)):
    raise RuntimeError("STEP 6.30H-0 must pass before STEP 6.30H.")

if bool(globals().get("FD004_GRU_ARCHITECTURE_LOCKED_630H", False)):
    raise RuntimeError("STEP 6.30H has already been completed.")

required_630h = [
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
    "FD004_ARCH_TRAIN_SOURCE_CANDIDATES_630H0",
    "FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0",
]

missing_630h = [
    name_630h
    for name_630h in required_630h
    if name_630h not in globals()
]

if missing_630h:
    raise RuntimeError(
        "Missing required STEP 6.30H input object(s): "
        + ", ".join(missing_630h)
    )


# =============================================================================
# 2) Verify recovered source-cell authority
# =============================================================================

source_candidates_630h = list(
    FD004_ARCH_TRAIN_SOURCE_CANDIDATES_630H0
)

arch_matches_630h = [
    row_630h
    for row_630h in source_candidates_630h
    if (
        "STEP 6.29H" in row_630h.get("priority_hits", [])
        and
        "FD003GRURegressor629H" in row_630h.get("priority_hits", [])
    )
]

train_matches_630h = [
    row_630h
    for row_630h in source_candidates_630h
    if (
        "STEP 6.29I" in row_630h.get("priority_hits", [])
        and
        "FD003_TRAINING_PROTOCOL" in row_630h.get("priority_hits", [])
    )
]

if len(arch_matches_630h) < 1:
    raise RuntimeError(
        "Recovered STEP 6.29H architecture source was not found."
    )

if len(train_matches_630h) < 1:
    raise RuntimeError(
        "Recovered STEP 6.29I training-protocol source was not found."
    )

arch_source_630h = max(
    arch_matches_630h,
    key=lambda row_630h: int(row_630h["score"]),
)

train_source_630h = max(
    train_matches_630h,
    key=lambda row_630h: int(row_630h["score"]),
)

EXPECTED_ARCH_SOURCE_SHA256_630H = (
    "5a5cf387e77fb455f43e58e264c0aed150007bf6367599953334ece8519b4446"
)

EXPECTED_TRAIN_SOURCE_SHA256_630H = (
    "b33087d23e568172593a10c148694526b7ff26876fe020628cb497687b169c0e"
)

if arch_source_630h["source_sha256"] != EXPECTED_ARCH_SOURCE_SHA256_630H:
    raise RuntimeError(
        "Recovered STEP 6.29H architecture source fingerprint changed."
    )

if train_source_630h["source_sha256"] != EXPECTED_TRAIN_SOURCE_SHA256_630H:
    raise RuntimeError(
        "Recovered STEP 6.29I training-protocol source fingerprint changed."
    )


# =============================================================================
# 3) Freeze FD004 architecture
# =============================================================================

FD004_GRU_INPUT_SIZE_630H = int(
    len(FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2)
)

FD004_GRU_HIDDEN_SIZE_630H = 64
FD004_GRU_NUM_LAYERS_630H = 1
FD004_GRU_DENSE_SIZE_630H = 32
FD004_GRU_DROPOUT_630H = 0.2
FD004_GRU_OUTPUT_SIZE_630H = 1

if FD004_GRU_INPUT_SIZE_630H != 24:
    raise RuntimeError(
        "FD004 locked feature contract does not expose exactly 24 model inputs."
    )


class FD004GRURegressor630H(nn.Module):

    def __init__(
        self,
        input_size=24,
        hidden_size=64,
        num_layers=1,
        dense_size=32,
        dropout=0.2,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc1 = nn.Linear(
            hidden_size,
            dense_size,
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc2 = nn.Linear(
            dense_size,
            1,
        )

    def forward(self, x):
        gru_output, _ = self.gru(x)
        last_hidden = gru_output[:, -1, :]
        x = self.fc1(last_hidden)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)


forward_names_630h = list(
    FD004GRURegressor630H.forward.__code__.co_names
)

expected_forward_names_630h = [
    "gru",
    "fc1",
    "relu",
    "dropout",
    "fc2",
    "squeeze",
]

if forward_names_630h != expected_forward_names_630h:
    raise RuntimeError(
        "FD004 forward operation sequence differs from the recovered contract."
    )

forward_opnames_630h = [
    instruction_630h.opname
    for instruction_630h
    in dis.get_instructions(
        FD004GRURegressor630H.forward
    )
]

if "BINARY_SUBSCR" not in forward_opnames_630h:
    raise RuntimeError(
        "FD004 forward semantics do not contain final-time-step indexing."
    )


FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H = {
    "gru.weight_ih_l0": (
        3 * FD004_GRU_HIDDEN_SIZE_630H,
        FD004_GRU_INPUT_SIZE_630H,
    ),
    "gru.weight_hh_l0": (
        3 * FD004_GRU_HIDDEN_SIZE_630H,
        FD004_GRU_HIDDEN_SIZE_630H,
    ),
    "gru.bias_ih_l0": (
        3 * FD004_GRU_HIDDEN_SIZE_630H,
    ),
    "gru.bias_hh_l0": (
        3 * FD004_GRU_HIDDEN_SIZE_630H,
    ),
    "fc1.weight": (
        FD004_GRU_DENSE_SIZE_630H,
        FD004_GRU_HIDDEN_SIZE_630H,
    ),
    "fc1.bias": (
        FD004_GRU_DENSE_SIZE_630H,
    ),
    "fc2.weight": (
        1,
        FD004_GRU_DENSE_SIZE_630H,
    ),
    "fc2.bias": (
        1,
    ),
}

FD004_GRU_EXPECTED_PARAMETER_COUNT_630H = int(
    (
        3
        * FD004_GRU_HIDDEN_SIZE_630H
        * FD004_GRU_INPUT_SIZE_630H
    )
    +
    (
        3
        * FD004_GRU_HIDDEN_SIZE_630H
        * FD004_GRU_HIDDEN_SIZE_630H
    )
    +
    (
        2
        * 3
        * FD004_GRU_HIDDEN_SIZE_630H
    )
    +
    (
        FD004_GRU_DENSE_SIZE_630H
        * FD004_GRU_HIDDEN_SIZE_630H
    )
    +
    FD004_GRU_DENSE_SIZE_630H
    +
    FD004_GRU_DENSE_SIZE_630H
    +
    1
)

if FD004_GRU_EXPECTED_PARAMETER_COUNT_630H != 19393:
    raise RuntimeError(
        "FD004 analytical GRU parameter count is not the expected 19,393."
    )


# =============================================================================
# 4) Freeze FD004 single-run training protocol
# =============================================================================

FD004_TRAINING_PROTOCOL_PROVENANCE_630H = (
    "cross-dataset transfer of the prospectively frozen FD003 STEP 6.29I "
    "single-run training protocol; no FD004 training or DEV model score "
    "was observed before this freeze"
)

FD004_TRAINING_SEED_630H = 42

FD004_TRAIN_BATCH_SIZE_630H = 256
FD004_EVAL_BATCH_SIZE_630H = 512

FD004_OPTIMIZER_630H = "Adam"
FD004_LEARNING_RATE_630H = 1e-3
FD004_ADAM_BETAS_630H = (0.9, 0.999)
FD004_ADAM_EPS_630H = 1e-8
FD004_WEIGHT_DECAY_630H = 0.0

FD004_TRAIN_LOSS_630H = "MSELoss"

FD004_MAX_EPOCHS_630H = 100
FD004_EARLY_STOPPING_MONITOR_630H = "DEV_MAE"
FD004_EARLY_STOPPING_MODE_630H = "min"
FD004_EARLY_STOPPING_PATIENCE_630H = 10
FD004_EARLY_STOPPING_MIN_DELTA_630H = 0.0
FD004_RESTORE_BEST_WEIGHTS_630H = True
FD004_TIE_BREAK_RULE_630H = "earliest epoch"

FD004_FIT_SHUFFLE_630H = True
FD004_SINGLE_TRAINING_RUN_630H = True
FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H = False

FD004_LR_SCHEDULER_630H = None
FD004_GRADIENT_CLIPPING_630H = None

FD004_RNG_PROTOCOL_630H = {
    "python_random_seed":
        FD004_TRAINING_SEED_630H,
    "numpy_seed":
        FD004_TRAINING_SEED_630H,
    "torch_cpu_seed":
        FD004_TRAINING_SEED_630H,
    "torch_cuda_seed_all_if_available":
        FD004_TRAINING_SEED_630H,
    "fit_dataloader_generator_seed":
        FD004_TRAINING_SEED_630H,
}


# =============================================================================
# 5) Freeze architecture contract
# =============================================================================

FD004_GRU_ARCHITECTURE_CONTRACT_630H = {
    "step":
        "6.30H",

    "dataset":
        "FD004",

    "transfer_family":
        "FD003GRURegressor629H_from_FD002GRURegressor628EB",

    "architecture_source_sha256":
        arch_source_630h["source_sha256"],

    "feature_contract_resolution_sha256":
        FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "dataset_specific_architecture_change":
        "input_size changed from FD003-locked 18 to FD004-locked 24 only",

    "input_size":
        FD004_GRU_INPUT_SIZE_630H,

    "hidden_size":
        FD004_GRU_HIDDEN_SIZE_630H,

    "num_layers":
        FD004_GRU_NUM_LAYERS_630H,

    "batch_first":
        True,

    "dense_size":
        FD004_GRU_DENSE_SIZE_630H,

    "activation":
        "ReLU",

    "dropout":
        FD004_GRU_DROPOUT_630H,

    "output_size":
        FD004_GRU_OUTPUT_SIZE_630H,

    "forward_semantics":
        "GRU full sequence -> final time-step output -> "
        "fc1 -> ReLU -> dropout -> fc2 -> squeeze(-1)",

    "expected_parameter_shapes": {
        key_630h:
            list(value_630h)
        for key_630h, value_630h
        in FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H.items()
    },

    "expected_parameter_count":
        FD004_GRU_EXPECTED_PARAMETER_COUNT_630H,

    "persistent_model_instantiated":
        False,

    "model_forward_executed":
        False,

    "model_training_performed":
        False,

    "dev_scoring_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

architecture_payload_630h = json.dumps(
    FD004_GRU_ARCHITECTURE_CONTRACT_630H,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H = hashlib.sha256(
    architecture_payload_630h
).hexdigest()


# =============================================================================
# 6) Freeze training-protocol contract
# =============================================================================

FD004_TRAINING_PROTOCOL_CONTRACT_630H = {
    "step":
        "6.30H",

    "dataset":
        "FD004",

    "provenance":
        FD004_TRAINING_PROTOCOL_PROVENANCE_630H,

    "training_protocol_source_sha256":
        train_source_630h["source_sha256"],

    "source_locator_sha256":
        FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0,

    "architecture_contract_sha256":
        FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "training_seed":
        FD004_TRAINING_SEED_630H,

    "train_batch_size":
        FD004_TRAIN_BATCH_SIZE_630H,

    "eval_batch_size":
        FD004_EVAL_BATCH_SIZE_630H,

    "optimizer":
        FD004_OPTIMIZER_630H,

    "learning_rate":
        FD004_LEARNING_RATE_630H,

    "adam_betas":
        list(FD004_ADAM_BETAS_630H),

    "adam_eps":
        FD004_ADAM_EPS_630H,

    "weight_decay":
        FD004_WEIGHT_DECAY_630H,

    "loss":
        FD004_TRAIN_LOSS_630H,

    "max_epochs":
        FD004_MAX_EPOCHS_630H,

    "early_stopping_monitor":
        FD004_EARLY_STOPPING_MONITOR_630H,

    "early_stopping_mode":
        FD004_EARLY_STOPPING_MODE_630H,

    "early_stopping_patience":
        FD004_EARLY_STOPPING_PATIENCE_630H,

    "early_stopping_min_delta":
        FD004_EARLY_STOPPING_MIN_DELTA_630H,

    "restore_best_weights":
        FD004_RESTORE_BEST_WEIGHTS_630H,

    "tie_break_rule":
        FD004_TIE_BREAK_RULE_630H,

    "fit_shuffle":
        FD004_FIT_SHUFFLE_630H,

    "single_training_run":
        FD004_SINGLE_TRAINING_RUN_630H,

    "dev_based_retuning_after_run_permitted":
        FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H,

    "lr_scheduler":
        FD004_LR_SCHEDULER_630H,

    "gradient_clipping":
        FD004_GRADIENT_CLIPPING_630H,

    "rng_protocol":
        dict(FD004_RNG_PROTOCOL_630H),

    "persistent_model_instantiated":
        False,

    "optimizer_created":
        False,

    "model_forward_executed":
        False,

    "optimizer_step_executed":
        False,

    "model_training_performed":
        False,

    "dev_model_score_observed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

training_payload_630h = json.dumps(
    FD004_TRAINING_PROTOCOL_CONTRACT_630H,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H = hashlib.sha256(
    training_payload_630h
).hexdigest()


# =============================================================================
# 7) Lock state
# =============================================================================

FD004_GRU_ARCHITECTURE_LOCKED_630H = True
FD004_TRAINING_PROTOCOL_LOCKED_630H = True

FD004_PERSISTENT_MODEL_INSTANCE_CREATED_630H = False
FD004_OPTIMIZER_CREATED_630H = False
FD004_MODEL_FORWARD_EXECUTED_630H = False
FD004_MODEL_TRAINING_PERFORMED_630H = False
FD004_DEV_SCORING_PERFORMED_630H = False
FD004_TEST_CONTENTS_READ_630H = False
FD004_RUL_CONTENTS_READ_630H = False
FD004_THRESHOLD_SELECTED_630H = False
FD004_POLICY_SELECTED_630H = False

all_checks_passed_630h = True


# =============================================================================
# 8) Report
# =============================================================================

print("=" * 160)

print(
    "STEP 6.30H — FD004 GRU ARCHITECTURE + "
    "SINGLE-RUN TRAINING-PROTOCOL PROSPECTIVE FREEZE"
)

print("=" * 160)

print("Persistent model instantiated :", False)
print("Optimizer created             :", False)
print("Model forward pass performed  :", False)
print("Model training performed      :", False)
print("DEV model scoring performed   :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)

print("\n--- RECOVERED SOURCE AUTHORITY ---")

print(
    "Architecture source SHA-256   :",
    arch_source_630h["source_sha256"],
)

print(
    "Training source SHA-256       :",
    train_source_630h["source_sha256"],
)

print("\n--- LOCKED FD004 GRU ARCHITECTURE ---")

print("Input size                    :", FD004_GRU_INPUT_SIZE_630H)
print("Hidden size                   :", FD004_GRU_HIDDEN_SIZE_630H)
print("GRU layers                    :", FD004_GRU_NUM_LAYERS_630H)
print("Dense size                    :", FD004_GRU_DENSE_SIZE_630H)
print("Dropout                       :", FD004_GRU_DROPOUT_630H)
print("Output size                   :", FD004_GRU_OUTPUT_SIZE_630H)

print(
    "Expected parameter count      :",
    FD004_GRU_EXPECTED_PARAMETER_COUNT_630H,
)

print(
    "Forward names exact           :",
    forward_names_630h == expected_forward_names_630h,
)

print(
    "Final-time-step indexing      :",
    "BINARY_SUBSCR" in forward_opnames_630h,
)

print("\n--- LOCKED SINGLE-RUN TRAINING PROTOCOL ---")

print("Training seed                 :", FD004_TRAINING_SEED_630H)
print("Training loss                 :", FD004_TRAIN_LOSS_630H)
print("Optimizer                     :", FD004_OPTIMIZER_630H)
print("Learning rate                 :", FD004_LEARNING_RATE_630H)
print("Adam betas                    :", FD004_ADAM_BETAS_630H)
print("Adam eps                      :", FD004_ADAM_EPS_630H)
print("Weight decay                  :", FD004_WEIGHT_DECAY_630H)
print("FIT batch size                :", FD004_TRAIN_BATCH_SIZE_630H)
print("Evaluation batch size         :", FD004_EVAL_BATCH_SIZE_630H)
print("FIT shuffle                   :", FD004_FIT_SHUFFLE_630H)
print("Maximum epochs                :", FD004_MAX_EPOCHS_630H)
print("Early-stop monitor            :", FD004_EARLY_STOPPING_MONITOR_630H)
print("Early-stop mode               :", FD004_EARLY_STOPPING_MODE_630H)
print("Early-stop patience           :", FD004_EARLY_STOPPING_PATIENCE_630H)
print("Early-stop minimum delta      :", FD004_EARLY_STOPPING_MIN_DELTA_630H)
print("Restore best weights          :", FD004_RESTORE_BEST_WEIGHTS_630H)
print("Tie-break rule                :", FD004_TIE_BREAK_RULE_630H)
print("LR scheduler                  :", FD004_LR_SCHEDULER_630H)
print("Gradient clipping             :", FD004_GRADIENT_CLIPPING_630H)
print("Single prospective run        :", FD004_SINGLE_TRAINING_RUN_630H)

print(
    "Post-run DEV retuning allowed :",
    FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H,
)

print("\n--- CONTRACT FINGERPRINTS ---")

print(
    "Architecture contract SHA-256 :",
    FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H,
)

print(
    "Training protocol SHA-256     :",
    FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H,
)

print(
    "\nPASS — the FD004 GRU architecture and single-run training protocol "
    "are prospectively frozen before persistent model initialization or training."
)

print(
    "Next: create deterministic FIT/DEV DataLoaders and the persistent FD004 "
    "model/loss/optimizer exactly once, fingerprint their initial state, and "
    "still stop before the first optimizer step."
)

In [ ]:
# STEP 6.30H-R1
# FD004 GRU ARCHITECTURE + SINGLE-RUN TRAINING-PROTOCOL
# PROSPECTIVE FREEZE — EXACT SOURCE-FINGERPRINT CORRECTION
#
# IMPORTANT:
#   - corrects only the source-resolution logic from failed STEP 6.30H
#   - no persistent model instance
#   - no optimizer
#   - no forward pass
#   - no training
#   - no DEV scoring
#   - no TEST / official RUL access

import hashlib
import json
import dis
import torch
import torch.nn as nn


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630g", False)):
    raise RuntimeError("STEP 6.30G must pass before STEP 6.30H-R1.")

if not bool(globals().get("all_checks_passed_630h0", False)):
    raise RuntimeError("STEP 6.30H-0 must pass before STEP 6.30H-R1.")

if bool(globals().get("FD004_GRU_ARCHITECTURE_LOCKED_630H", False)):
    raise RuntimeError(
        "FD004 STEP 6.30H architecture is already locked; do not run again."
    )

required_630h = [
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
    "FD004_ARCH_TRAIN_SOURCE_CANDIDATES_630H0",
    "FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0",
]

missing_630h = [
    name_630h
    for name_630h in required_630h
    if name_630h not in globals()
]

if missing_630h:
    raise RuntimeError(
        "Missing required STEP 6.30H-R1 input object(s): "
        + ", ".join(missing_630h)
    )


# =============================================================================
# 2) Exact source-fingerprint resolution
# =============================================================================

EXPECTED_ARCH_SOURCE_SHA256_630H = (
    "5a5cf387e77fb455f43e58e264c0aed150007bf6367599953334ece8519b4446"
)

EXPECTED_TRAIN_SOURCE_SHA256_630H = (
    "b33087d23e568172593a10c148694526b7ff26876fe020628cb497687b169c0e"
)

source_candidates_630h = list(
    FD004_ARCH_TRAIN_SOURCE_CANDIDATES_630H0
)

arch_exact_matches_630h = [
    row_630h
    for row_630h in source_candidates_630h
    if str(row_630h.get("source_sha256", ""))
    == EXPECTED_ARCH_SOURCE_SHA256_630H
]

train_exact_matches_630h = [
    row_630h
    for row_630h in source_candidates_630h
    if str(row_630h.get("source_sha256", ""))
    == EXPECTED_TRAIN_SOURCE_SHA256_630H
]

if len(arch_exact_matches_630h) != 1:
    raise RuntimeError(
        "Could not uniquely resolve the frozen STEP 6.29H "
        "architecture source by exact SHA-256."
    )

if len(train_exact_matches_630h) != 1:
    raise RuntimeError(
        "Could not uniquely resolve the frozen STEP 6.29I "
        "training-protocol source by exact SHA-256."
    )

arch_source_630h = arch_exact_matches_630h[0]
train_source_630h = train_exact_matches_630h[0]

if int(arch_source_630h.get("cell_index", -1)) != 171:
    raise RuntimeError(
        "Resolved architecture source is not expected cell 171."
    )

if int(train_source_630h.get("cell_index", -1)) != 172:
    raise RuntimeError(
        "Resolved training-protocol source is not expected cell 172."
    )


# =============================================================================
# 3) Freeze FD004 architecture
# =============================================================================

FD004_GRU_INPUT_SIZE_630H = int(
    len(FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2)
)

FD004_GRU_HIDDEN_SIZE_630H = 64
FD004_GRU_NUM_LAYERS_630H = 1
FD004_GRU_DENSE_SIZE_630H = 32
FD004_GRU_DROPOUT_630H = 0.2
FD004_GRU_OUTPUT_SIZE_630H = 1

if FD004_GRU_INPUT_SIZE_630H != 24:
    raise RuntimeError(
        "FD004 locked feature contract does not expose exactly 24 model inputs."
    )


class FD004GRURegressor630H(nn.Module):

    def __init__(
        self,
        input_size=24,
        hidden_size=64,
        num_layers=1,
        dense_size=32,
        dropout=0.2,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc1 = nn.Linear(
            hidden_size,
            dense_size,
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc2 = nn.Linear(
            dense_size,
            1,
        )

    def forward(self, x):
        gru_output, _ = self.gru(x)
        last_hidden = gru_output[:, -1, :]
        x = self.fc1(last_hidden)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)


forward_names_630h = list(
    FD004GRURegressor630H.forward.__code__.co_names
)

expected_forward_names_630h = [
    "gru",
    "fc1",
    "relu",
    "dropout",
    "fc2",
    "squeeze",
]

if forward_names_630h != expected_forward_names_630h:
    raise RuntimeError(
        "FD004 forward operation sequence differs from recovered semantics."
    )

forward_opnames_630h = [
    instruction_630h.opname
    for instruction_630h
    in dis.get_instructions(
        FD004GRURegressor630H.forward
    )
]

if "BINARY_SUBSCR" not in forward_opnames_630h:
    raise RuntimeError(
        "FD004 forward semantics do not contain final-time-step indexing."
    )


FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H = {
    "gru.weight_ih_l0": (192, 24),
    "gru.weight_hh_l0": (192, 64),
    "gru.bias_ih_l0": (192,),
    "gru.bias_hh_l0": (192,),
    "fc1.weight": (32, 64),
    "fc1.bias": (32,),
    "fc2.weight": (1, 32),
    "fc2.bias": (1,),
}

FD004_GRU_EXPECTED_PARAMETER_COUNT_630H = 19393


# =============================================================================
# 4) Freeze FD004 prospective single-run training protocol
# =============================================================================

FD004_TRAINING_PROTOCOL_PROVENANCE_630H = (
    "cross-dataset transfer of the prospectively frozen FD003 STEP 6.29I "
    "single-run training protocol; no FD004 training or DEV model score "
    "was observed before this freeze"
)

FD004_TRAINING_SEED_630H = 42

FD004_TRAIN_BATCH_SIZE_630H = 256
FD004_EVAL_BATCH_SIZE_630H = 512

FD004_OPTIMIZER_630H = "Adam"
FD004_LEARNING_RATE_630H = 1e-3
FD004_ADAM_BETAS_630H = (0.9, 0.999)
FD004_ADAM_EPS_630H = 1e-8
FD004_WEIGHT_DECAY_630H = 0.0

FD004_TRAIN_LOSS_630H = "MSELoss"

FD004_MAX_EPOCHS_630H = 100
FD004_EARLY_STOPPING_MONITOR_630H = "DEV_MAE"
FD004_EARLY_STOPPING_MODE_630H = "min"
FD004_EARLY_STOPPING_PATIENCE_630H = 10
FD004_EARLY_STOPPING_MIN_DELTA_630H = 0.0
FD004_RESTORE_BEST_WEIGHTS_630H = True
FD004_TIE_BREAK_RULE_630H = "earliest epoch"

FD004_FIT_SHUFFLE_630H = True
FD004_SINGLE_TRAINING_RUN_630H = True
FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H = False

FD004_LR_SCHEDULER_630H = None
FD004_GRADIENT_CLIPPING_630H = None

FD004_RNG_PROTOCOL_630H = {
    "python_random_seed":
        FD004_TRAINING_SEED_630H,
    "numpy_seed":
        FD004_TRAINING_SEED_630H,
    "torch_cpu_seed":
        FD004_TRAINING_SEED_630H,
    "torch_cuda_seed_all_if_available":
        FD004_TRAINING_SEED_630H,
    "fit_dataloader_generator_seed":
        FD004_TRAINING_SEED_630H,
}


# =============================================================================
# 5) Architecture contract
# =============================================================================

FD004_GRU_ARCHITECTURE_CONTRACT_630H = {
    "step":
        "6.30H",

    "dataset":
        "FD004",

    "transfer_family":
        "FD003GRURegressor629H_from_FD002GRURegressor628EB",

    "architecture_source_sha256":
        arch_source_630h["source_sha256"],

    "feature_contract_resolution_sha256":
        FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "dataset_specific_architecture_change":
        "input_size changed from FD003-locked 18 to FD004-locked 24 only",

    "input_size":
        FD004_GRU_INPUT_SIZE_630H,

    "hidden_size":
        FD004_GRU_HIDDEN_SIZE_630H,

    "num_layers":
        FD004_GRU_NUM_LAYERS_630H,

    "batch_first":
        True,

    "dense_size":
        FD004_GRU_DENSE_SIZE_630H,

    "activation":
        "ReLU",

    "dropout":
        FD004_GRU_DROPOUT_630H,

    "output_size":
        FD004_GRU_OUTPUT_SIZE_630H,

    "forward_semantics":
        "GRU full sequence -> final time-step output -> "
        "fc1 -> ReLU -> dropout -> fc2 -> squeeze(-1)",

    "expected_parameter_shapes": {
        key_630h:
            list(value_630h)
        for key_630h, value_630h
        in FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H.items()
    },

    "expected_parameter_count":
        FD004_GRU_EXPECTED_PARAMETER_COUNT_630H,

    "persistent_model_instantiated":
        False,

    "model_forward_executed":
        False,

    "model_training_performed":
        False,

    "dev_scoring_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

architecture_payload_630h = json.dumps(
    FD004_GRU_ARCHITECTURE_CONTRACT_630H,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H = hashlib.sha256(
    architecture_payload_630h
).hexdigest()


# =============================================================================
# 6) Training-protocol contract
# =============================================================================

FD004_TRAINING_PROTOCOL_CONTRACT_630H = {
    "step":
        "6.30H",

    "dataset":
        "FD004",

    "provenance":
        FD004_TRAINING_PROTOCOL_PROVENANCE_630H,

    "training_protocol_source_sha256":
        train_source_630h["source_sha256"],

    "source_locator_sha256":
        FD004_ARCH_TRAIN_SOURCE_LOCATOR_SHA256_630H0,

    "architecture_contract_sha256":
        FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "training_seed":
        FD004_TRAINING_SEED_630H,

    "train_batch_size":
        FD004_TRAIN_BATCH_SIZE_630H,

    "eval_batch_size":
        FD004_EVAL_BATCH_SIZE_630H,

    "optimizer":
        FD004_OPTIMIZER_630H,

    "learning_rate":
        FD004_LEARNING_RATE_630H,

    "adam_betas":
        list(FD004_ADAM_BETAS_630H),

    "adam_eps":
        FD004_ADAM_EPS_630H,

    "weight_decay":
        FD004_WEIGHT_DECAY_630H,

    "loss":
        FD004_TRAIN_LOSS_630H,

    "max_epochs":
        FD004_MAX_EPOCHS_630H,

    "early_stopping_monitor":
        FD004_EARLY_STOPPING_MONITOR_630H,

    "early_stopping_mode":
        FD004_EARLY_STOPPING_MODE_630H,

    "early_stopping_patience":
        FD004_EARLY_STOPPING_PATIENCE_630H,

    "early_stopping_min_delta":
        FD004_EARLY_STOPPING_MIN_DELTA_630H,

    "restore_best_weights":
        FD004_RESTORE_BEST_WEIGHTS_630H,

    "tie_break_rule":
        FD004_TIE_BREAK_RULE_630H,

    "fit_shuffle":
        FD004_FIT_SHUFFLE_630H,

    "single_training_run":
        FD004_SINGLE_TRAINING_RUN_630H,

    "dev_based_retuning_after_run_permitted":
        FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H,

    "lr_scheduler":
        FD004_LR_SCHEDULER_630H,

    "gradient_clipping":
        FD004_GRADIENT_CLIPPING_630H,

    "rng_protocol":
        dict(FD004_RNG_PROTOCOL_630H),

    "persistent_model_instantiated":
        False,

    "optimizer_created":
        False,

    "model_forward_executed":
        False,

    "optimizer_step_executed":
        False,

    "model_training_performed":
        False,

    "dev_model_score_observed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

training_payload_630h = json.dumps(
    FD004_TRAINING_PROTOCOL_CONTRACT_630H,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H = hashlib.sha256(
    training_payload_630h
).hexdigest()


# =============================================================================
# 7) Final lock state
# =============================================================================

FD004_GRU_ARCHITECTURE_LOCKED_630H = True
FD004_TRAINING_PROTOCOL_LOCKED_630H = True

FD004_PERSISTENT_MODEL_INSTANCE_CREATED_630H = False
FD004_OPTIMIZER_CREATED_630H = False
FD004_MODEL_FORWARD_EXECUTED_630H = False
FD004_MODEL_TRAINING_PERFORMED_630H = False
FD004_DEV_SCORING_PERFORMED_630H = False

FD004_TEST_CONTENTS_READ_630H = False
FD004_RUL_CONTENTS_READ_630H = False

FD004_THRESHOLD_SELECTED_630H = False
FD004_POLICY_SELECTED_630H = False

all_checks_passed_630h = True


# =============================================================================
# 8) Report
# =============================================================================

print("=" * 160)

print(
    "STEP 6.30H-R1 — FD004 GRU ARCHITECTURE + "
    "SINGLE-RUN TRAINING-PROTOCOL PROSPECTIVE FREEZE"
)

print("=" * 160)

print(
    "Prior failure cause            :",
    "locator ranking selected a reference cell",
)

print("Persistent model instantiated :", False)
print("Optimizer created             :", False)
print("Model forward pass performed  :", False)
print("Model training performed      :", False)
print("DEV model scoring performed   :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)

print("\n--- EXACT RECOVERED SOURCE AUTHORITY ---")

print(
    "Architecture cell index       :",
    arch_source_630h["cell_index"],
)

print(
    "Architecture source SHA-256   :",
    arch_source_630h["source_sha256"],
)

print(
    "Training cell index           :",
    train_source_630h["cell_index"],
)

print(
    "Training source SHA-256       :",
    train_source_630h["source_sha256"],
)

print("\n--- LOCKED FD004 GRU ARCHITECTURE ---")

print("Input size                    :", FD004_GRU_INPUT_SIZE_630H)
print("Hidden size                   :", FD004_GRU_HIDDEN_SIZE_630H)
print("GRU layers                    :", FD004_GRU_NUM_LAYERS_630H)
print("Dense size                    :", FD004_GRU_DENSE_SIZE_630H)
print("Dropout                       :", FD004_GRU_DROPOUT_630H)
print("Output size                   :", FD004_GRU_OUTPUT_SIZE_630H)

print(
    "Expected parameter count      :",
    FD004_GRU_EXPECTED_PARAMETER_COUNT_630H,
)

print(
    "Forward names exact           :",
    forward_names_630h == expected_forward_names_630h,
)

print(
    "Final-time-step indexing      :",
    "BINARY_SUBSCR" in forward_opnames_630h,
)

print("\n--- LOCKED SINGLE-RUN TRAINING PROTOCOL ---")

print("Training seed                 :", FD004_TRAINING_SEED_630H)
print("Training loss                 :", FD004_TRAIN_LOSS_630H)
print("Optimizer                     :", FD004_OPTIMIZER_630H)
print("Learning rate                 :", FD004_LEARNING_RATE_630H)
print("Adam betas                    :", FD004_ADAM_BETAS_630H)
print("Adam eps                      :", FD004_ADAM_EPS_630H)
print("Weight decay                  :", FD004_WEIGHT_DECAY_630H)
print("FIT batch size                :", FD004_TRAIN_BATCH_SIZE_630H)
print("Evaluation batch size         :", FD004_EVAL_BATCH_SIZE_630H)
print("FIT shuffle                   :", FD004_FIT_SHUFFLE_630H)
print("Maximum epochs                :", FD004_MAX_EPOCHS_630H)

print(
    "Early-stop monitor            :",
    FD004_EARLY_STOPPING_MONITOR_630H,
)

print(
    "Early-stop mode               :",
    FD004_EARLY_STOPPING_MODE_630H,
)

print(
    "Early-stop patience           :",
    FD004_EARLY_STOPPING_PATIENCE_630H,
)

print(
    "Early-stop minimum delta      :",
    FD004_EARLY_STOPPING_MIN_DELTA_630H,
)

print(
    "Restore best weights          :",
    FD004_RESTORE_BEST_WEIGHTS_630H,
)

print(
    "Tie-break rule                :",
    FD004_TIE_BREAK_RULE_630H,
)

print("LR scheduler                  :", FD004_LR_SCHEDULER_630H)
print("Gradient clipping             :", FD004_GRADIENT_CLIPPING_630H)

print(
    "Single prospective run        :",
    FD004_SINGLE_TRAINING_RUN_630H,
)

print(
    "Post-run DEV retuning allowed :",
    FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H,
)

print("\n--- CONTRACT FINGERPRINTS ---")

print(
    "Architecture contract SHA-256 :",
    FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H,
)

print(
    "Training protocol SHA-256     :",
    FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H,
)

print(
    "\nPASS — exact source fingerprints resolved the locator ambiguity. "
    "The FD004 architecture and single-run training protocol are now "
    "prospectively frozen before initialization or training."
)

print(
    "Next: create deterministic FIT/DEV DataLoaders and the persistent FD004 "
    "model/loss/optimizer exactly once, fingerprint their initial state, "
    "and stop before the first optimizer step."
)

In [ ]:
# STEP 6.30I
# FD004 DETERMINISTIC DATA-PIPELINE + PERSISTENT MODEL/LOSS/OPTIMIZER INITIALIZATION LOCK
#
# INITIALIZATION ONLY:
#   - create writable tensor-backed FIT/DEV datasets
#   - create deterministic DataLoaders
#   - instantiate persistent model exactly once
#   - create loss and Adam optimizer exactly once
#   - fingerprint initial state
#   - NO DataLoader iteration
#   - NO forward pass
#   - NO backward pass
#   - NO optimizer step
#   - NO DEV scoring
#   - NO TEST / official RUL access

import hashlib
import json
import random
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630i(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30I hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_tensor_630i(value):
    tensor = value.detach().cpu().contiguous()
    arr = tensor.numpy()
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_model_state_630i(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()

        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630i(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630h", False)):
    raise RuntimeError("STEP 6.30H-R1 must pass before STEP 6.30I.")

if not bool(globals().get("FD004_GRU_ARCHITECTURE_LOCKED_630H", False)):
    raise RuntimeError("FD004 architecture is not frozen.")

if not bool(globals().get("FD004_TRAINING_PROTOCOL_LOCKED_630H", False)):
    raise RuntimeError("FD004 training protocol is not frozen.")

if bool(globals().get("FD004_PERSISTENT_MODEL_INSTANCE_CREATED_630I", False)):
    raise RuntimeError("STEP 6.30I persistent model was already created.")

required_630i = [
    "X_FIT_FD004_630G",
    "y_FIT_FD004_630G",
    "X_DEV_FD004_630G",
    "y_DEV_FD004_630G",
    "FD004_X_FIT_SHA256_630G",
    "FD004_Y_FIT_SHA256_630G",
    "FD004_X_DEV_SHA256_630G",
    "FD004_Y_DEV_SHA256_630G",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
    "FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H",
    "FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H",
    "FD004_GRU_INPUT_SIZE_630H",
    "FD004_GRU_HIDDEN_SIZE_630H",
    "FD004_GRU_NUM_LAYERS_630H",
    "FD004_GRU_DENSE_SIZE_630H",
    "FD004_GRU_DROPOUT_630H",
    "FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H",
    "FD004_GRU_EXPECTED_PARAMETER_COUNT_630H",
    "FD004_TRAINING_SEED_630H",
    "FD004_TRAIN_BATCH_SIZE_630H",
    "FD004_EVAL_BATCH_SIZE_630H",
    "FD004_LEARNING_RATE_630H",
    "FD004_ADAM_BETAS_630H",
    "FD004_ADAM_EPS_630H",
    "FD004_WEIGHT_DECAY_630H",
    "FD004_FIT_SHUFFLE_630H",
    "FD004_MAX_EPOCHS_630H",
    "FD004_EARLY_STOPPING_PATIENCE_630H",
    "FD004_EARLY_STOPPING_MIN_DELTA_630H",
    "FD004_RESTORE_BEST_WEIGHTS_630H",
    "FD004_TIE_BREAK_RULE_630H",
    "FD004GRURegressor630H",
]

missing_630i = [name for name in required_630i if name not in globals()]

if missing_630i:
    raise RuntimeError(
        "Missing required STEP 6.30I object(s): "
        + ", ".join(missing_630i)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630H", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30I.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630H", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30I.")


# =============================================================================
# 3) Re-certify frozen window arrays
# =============================================================================

if _sha_arr_630i(X_FIT_FD004_630G) != FD004_X_FIT_SHA256_630G:
    raise RuntimeError("Frozen FD004 FIT window tensor changed.")

if _sha_arr_630i(y_FIT_FD004_630G) != FD004_Y_FIT_SHA256_630G:
    raise RuntimeError("Frozen FD004 FIT target vector changed.")

if _sha_arr_630i(X_DEV_FD004_630G) != FD004_X_DEV_SHA256_630G:
    raise RuntimeError("Frozen FD004 DEV window tensor changed.")

if _sha_arr_630i(y_DEV_FD004_630G) != FD004_Y_DEV_SHA256_630G:
    raise RuntimeError("Frozen FD004 DEV target vector changed.")


# =============================================================================
# 4) Establish deterministic RNG state BEFORE model initialization
# =============================================================================

seed_630i = int(FD004_TRAINING_SEED_630H)

random.seed(seed_630i)
np.random.seed(seed_630i)
torch.manual_seed(seed_630i)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_630i)

FD004_DEVICE_630I = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# =============================================================================
# 5) Create writable private tensor backings
# =============================================================================

fit_x_private_630i = np.array(
    X_FIT_FD004_630G,
    dtype=np.float32,
    copy=True,
    order="C",
)

fit_y_private_630i = np.array(
    y_FIT_FD004_630G,
    dtype=np.float32,
    copy=True,
    order="C",
)

dev_x_private_630i = np.array(
    X_DEV_FD004_630G,
    dtype=np.float32,
    copy=True,
    order="C",
)

dev_y_private_630i = np.array(
    y_DEV_FD004_630G,
    dtype=np.float32,
    copy=True,
    order="C",
)

if not (
    fit_x_private_630i.flags.writeable
    and fit_y_private_630i.flags.writeable
    and dev_x_private_630i.flags.writeable
    and dev_y_private_630i.flags.writeable
):
    raise RuntimeError("Private DataLoader backing arrays are not writable.")

if _sha_arr_630i(fit_x_private_630i) != FD004_X_FIT_SHA256_630G:
    raise RuntimeError("Writable FIT X copy differs from frozen evidence.")

if _sha_arr_630i(fit_y_private_630i) != FD004_Y_FIT_SHA256_630G:
    raise RuntimeError("Writable FIT y copy differs from frozen evidence.")

if _sha_arr_630i(dev_x_private_630i) != FD004_X_DEV_SHA256_630G:
    raise RuntimeError("Writable DEV X copy differs from frozen evidence.")

if _sha_arr_630i(dev_y_private_630i) != FD004_Y_DEV_SHA256_630G:
    raise RuntimeError("Writable DEV y copy differs from frozen evidence.")


FD004_X_FIT_TENSOR_630I = torch.from_numpy(fit_x_private_630i)
FD004_Y_FIT_TENSOR_630I = torch.from_numpy(fit_y_private_630i)

FD004_X_DEV_TENSOR_630I = torch.from_numpy(dev_x_private_630i)
FD004_Y_DEV_TENSOR_630I = torch.from_numpy(dev_y_private_630i)


# =============================================================================
# 6) Tensor identity checks
# =============================================================================

if tuple(FD004_X_FIT_TENSOR_630I.shape) != tuple(X_FIT_FD004_630G.shape):
    raise RuntimeError("FIT tensor shape changed.")

if tuple(FD004_X_DEV_TENSOR_630I.shape) != tuple(X_DEV_FD004_630G.shape):
    raise RuntimeError("DEV tensor shape changed.")

if FD004_X_FIT_TENSOR_630I.dtype != torch.float32:
    raise RuntimeError("FIT X tensor dtype is not float32.")

if FD004_Y_FIT_TENSOR_630I.dtype != torch.float32:
    raise RuntimeError("FIT y tensor dtype is not float32.")

if FD004_X_DEV_TENSOR_630I.dtype != torch.float32:
    raise RuntimeError("DEV X tensor dtype is not float32.")

if FD004_Y_DEV_TENSOR_630I.dtype != torch.float32:
    raise RuntimeError("DEV y tensor dtype is not float32.")

if _sha_tensor_630i(FD004_X_FIT_TENSOR_630I) != FD004_X_FIT_SHA256_630G:
    raise RuntimeError("FIT X tensor byte identity failed.")

if _sha_tensor_630i(FD004_Y_FIT_TENSOR_630I) != FD004_Y_FIT_SHA256_630G:
    raise RuntimeError("FIT y tensor byte identity failed.")

if _sha_tensor_630i(FD004_X_DEV_TENSOR_630I) != FD004_X_DEV_SHA256_630G:
    raise RuntimeError("DEV X tensor byte identity failed.")

if _sha_tensor_630i(FD004_Y_DEV_TENSOR_630I) != FD004_Y_DEV_SHA256_630G:
    raise RuntimeError("DEV y tensor byte identity failed.")


# =============================================================================
# 7) Create datasets + deterministic DataLoaders
# =============================================================================

FD004_FIT_DATASET_630I = TensorDataset(
    FD004_X_FIT_TENSOR_630I,
    FD004_Y_FIT_TENSOR_630I,
)

FD004_DEV_DATASET_630I = TensorDataset(
    FD004_X_DEV_TENSOR_630I,
    FD004_Y_DEV_TENSOR_630I,
)

FD004_FIT_LOADER_GENERATOR_630I = torch.Generator()
FD004_FIT_LOADER_GENERATOR_630I.manual_seed(seed_630i)

FD004_FIT_LOADER_630I = DataLoader(
    FD004_FIT_DATASET_630I,
    batch_size=int(FD004_TRAIN_BATCH_SIZE_630H),
    shuffle=bool(FD004_FIT_SHUFFLE_630H),
    generator=FD004_FIT_LOADER_GENERATOR_630I,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

FD004_DEV_LOADER_630I = DataLoader(
    FD004_DEV_DATASET_630I,
    batch_size=int(FD004_EVAL_BATCH_SIZE_630H),
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

fit_generator_state_sha_630i = _sha_tensor_630i(
    FD004_FIT_LOADER_GENERATOR_630I.get_state()
)


# =============================================================================
# 8) Instantiate persistent model exactly once
# =============================================================================

MODEL_FD004_630I = FD004GRURegressor630H(
    input_size=int(FD004_GRU_INPUT_SIZE_630H),
    hidden_size=int(FD004_GRU_HIDDEN_SIZE_630H),
    num_layers=int(FD004_GRU_NUM_LAYERS_630H),
    dense_size=int(FD004_GRU_DENSE_SIZE_630H),
    dropout=float(FD004_GRU_DROPOUT_630H),
)

actual_shapes_630i = {
    name: tuple(parameter.shape)
    for name, parameter in MODEL_FD004_630I.named_parameters()
}

if actual_shapes_630i != FD004_GRU_EXPECTED_PARAMETER_SHAPES_630H:
    raise RuntimeError(
        "Instantiated FD004 parameter shapes differ from frozen architecture."
    )

actual_parameter_count_630i = int(
    sum(
        parameter.numel()
        for parameter in MODEL_FD004_630I.parameters()
    )
)

if actual_parameter_count_630i != int(
    FD004_GRU_EXPECTED_PARAMETER_COUNT_630H
):
    raise RuntimeError(
        "Instantiated FD004 parameter count differs from frozen architecture."
    )

initial_model_sha_before_device_630i = _sha_model_state_630i(
    MODEL_FD004_630I
)

MODEL_FD004_630I = MODEL_FD004_630I.to(
    FD004_DEVICE_630I
)

initial_model_sha_after_device_630i = _sha_model_state_630i(
    MODEL_FD004_630I
)

if initial_model_sha_before_device_630i != initial_model_sha_after_device_630i:
    raise RuntimeError(
        "Moving the initialized model to the selected device changed parameters."
    )


# =============================================================================
# 9) Create persistent loss + optimizer exactly once
# =============================================================================

LOSS_FD004_630I = torch.nn.MSELoss()

OPTIMIZER_FD004_630I = torch.optim.Adam(
    MODEL_FD004_630I.parameters(),
    lr=float(FD004_LEARNING_RATE_630H),
    betas=tuple(FD004_ADAM_BETAS_630H),
    eps=float(FD004_ADAM_EPS_630H),
    weight_decay=float(FD004_WEIGHT_DECAY_630H),
)

model_sha_after_optimizer_creation_630i = _sha_model_state_630i(
    MODEL_FD004_630I
)

if model_sha_after_optimizer_creation_630i != initial_model_sha_after_device_630i:
    raise RuntimeError("Optimizer creation changed the initialized model state.")

if len(OPTIMIZER_FD004_630I.state) != 0:
    raise RuntimeError(
        "Adam optimizer state is unexpectedly non-empty before any optimizer step."
    )


# =============================================================================
# 10) Freeze initialization fingerprints
# =============================================================================

FD004_INITIAL_MODEL_SHA256_630I = (
    initial_model_sha_after_device_630i
)

FD004_FIT_LOADER_GENERATOR_STATE_SHA256_630I = (
    fit_generator_state_sha_630i
)

optimizer_config_630i = {
    "class":
        type(OPTIMIZER_FD004_630I).__name__,
    "lr":
        float(OPTIMIZER_FD004_630I.param_groups[0]["lr"]),
    "betas":
        [
            float(x)
            for x in OPTIMIZER_FD004_630I.param_groups[0]["betas"]
        ],
    "eps":
        float(OPTIMIZER_FD004_630I.param_groups[0]["eps"]),
    "weight_decay":
        float(OPTIMIZER_FD004_630I.param_groups[0]["weight_decay"]),
    "amsgrad":
        bool(OPTIMIZER_FD004_630I.param_groups[0]["amsgrad"]),
    "initial_state_entry_count":
        int(len(OPTIMIZER_FD004_630I.state)),
}

FD004_INITIAL_OPTIMIZER_CONFIG_SHA256_630I = _sha_json_630i(
    optimizer_config_630i
)


# =============================================================================
# 11) Initialization contract
# =============================================================================

FD004_INITIALIZATION_CONTRACT_630I = {
    "step":
        "6.30I",
    "dataset":
        "FD004",
    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,
    "architecture_contract_sha256":
        FD004_GRU_ARCHITECTURE_CONTRACT_SHA256_630H,
    "training_protocol_sha256":
        FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H,
    "seed":
        seed_630i,
    "device":
        str(FD004_DEVICE_630I),
    "fit_dataset_size":
        int(len(FD004_FIT_DATASET_630I)),
    "dev_dataset_size":
        int(len(FD004_DEV_DATASET_630I)),
    "fit_batch_size":
        int(FD004_FIT_LOADER_630I.batch_size),
    "dev_batch_size":
        int(FD004_DEV_LOADER_630I.batch_size),
    "fit_shuffle":
        True,
    "dev_shuffle":
        False,
    "num_workers":
        0,
    "pin_memory":
        False,
    "drop_last":
        False,
    "private_writable_backing":
        True,
    "fit_x_byte_identity":
        True,
    "fit_y_byte_identity":
        True,
    "dev_x_byte_identity":
        True,
    "dev_y_byte_identity":
        True,
    "model_parameter_count":
        actual_parameter_count_630i,
    "initial_model_sha256":
        FD004_INITIAL_MODEL_SHA256_630I,
    "fit_loader_generator_state_sha256":
        FD004_FIT_LOADER_GENERATOR_STATE_SHA256_630I,
    "optimizer_config_sha256":
        FD004_INITIAL_OPTIMIZER_CONFIG_SHA256_630I,
    "optimizer_state_entry_count":
        int(len(OPTIMIZER_FD004_630I.state)),
    "dataloader_iteration_performed":
        False,
    "model_forward_executed":
        False,
    "backward_executed":
        False,
    "optimizer_step_executed":
        False,
    "dev_scoring_performed":
        False,
    "model_training_performed":
        False,
    "test_contents_read":
        False,
    "official_rul_contents_read":
        False,
}

FD004_INITIALIZATION_CONTRACT_SHA256_630I = _sha_json_630i(
    FD004_INITIALIZATION_CONTRACT_630I
)


# =============================================================================
# 12) Lock state
# =============================================================================

FD004_PERSISTENT_MODEL_INSTANCE_CREATED_630I = True
FD004_LOSS_CREATED_630I = True
FD004_OPTIMIZER_CREATED_630I = True
FD004_DATALOADERS_CREATED_630I = True

FD004_DATALOADER_ITERATION_PERFORMED_630I = False
FD004_MODEL_FORWARD_EXECUTED_630I = False
FD004_BACKWARD_EXECUTED_630I = False
FD004_OPTIMIZER_STEP_EXECUTED_630I = False
FD004_DEV_SCORING_PERFORMED_630I = False
FD004_MODEL_TRAINING_PERFORMED_630I = False

FD004_TEST_CONTENTS_READ_630I = False
FD004_RUL_CONTENTS_READ_630I = False
FD004_THRESHOLD_SELECTED_630I = False
FD004_POLICY_SELECTED_630I = False

all_checks_passed_630i = True


# =============================================================================
# 13) Report
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30I — FD004 DETERMINISTIC DATA-PIPELINE + "
    "PERSISTENT MODEL/LOSS/OPTIMIZER INITIALIZATION LOCK"
)

print("=" * 168)

print("DataLoaders created             :", True)
print("Persistent model created       :", True)
print("Loss created                   :", True)
print("Optimizer created              :", True)
print("DataLoader iteration performed :", False)
print("Model forward pass performed   :", False)
print("Backward pass performed        :", False)
print("Optimizer step performed       :", False)
print("DEV model scoring performed    :", False)
print("Model training performed       :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- DATA PIPELINE ---")

print(
    "FIT dataset / batch size       :",
    (
        len(FD004_FIT_DATASET_630I),
        FD004_FIT_LOADER_630I.batch_size,
    ),
)

print(
    "DEV dataset / batch size       :",
    (
        len(FD004_DEV_DATASET_630I),
        FD004_DEV_LOADER_630I.batch_size,
    ),
)

print("FIT shuffle                    :", True)
print("DEV shuffle                    :", False)
print("Private writable backing       :", True)

print(
    "Tensor byte identity exact     :",
    (
        _sha_tensor_630i(FD004_X_FIT_TENSOR_630I)
        == FD004_X_FIT_SHA256_630G,
        _sha_tensor_630i(FD004_Y_FIT_TENSOR_630I)
        == FD004_Y_FIT_SHA256_630G,
        _sha_tensor_630i(FD004_X_DEV_TENSOR_630I)
        == FD004_X_DEV_SHA256_630G,
        _sha_tensor_630i(FD004_Y_DEV_TENSOR_630I)
        == FD004_Y_DEV_SHA256_630G,
    ),
)

print("\n--- PERSISTENT MODEL / OPTIMIZER ---")

print("Selected device                :", FD004_DEVICE_630I)
print("Model class                    :", type(MODEL_FD004_630I).__name__)
print("Model training mode            :", MODEL_FD004_630I.training)
print("Parameter count                :", actual_parameter_count_630i)
print("Optimizer                      :", type(OPTIMIZER_FD004_630I).__name__)
print("Optimizer state entries        :", len(OPTIMIZER_FD004_630I.state))

print("\n--- INITIAL FINGERPRINTS ---")

print(
    "Initial model SHA-256          :",
    FD004_INITIAL_MODEL_SHA256_630I,
)

print(
    "FIT loader generator SHA-256   :",
    FD004_FIT_LOADER_GENERATOR_STATE_SHA256_630I,
)

print(
    "Optimizer config SHA-256       :",
    FD004_INITIAL_OPTIMIZER_CONFIG_SHA256_630I,
)

print(
    "Initialization contract SHA-256:",
    FD004_INITIALIZATION_CONTRACT_SHA256_630I,
)

print(
    "\nPASS — deterministic FIT/DEV loaders and the persistent FD004 "
    "model/loss/optimizer were initialized exactly once and fingerprinted "
    "before any DataLoader iteration, forward pass, backward pass, or optimizer step."
)

print(
    "Next: execute the one prospectively frozen FD004 training run with "
    "DEV-MAE early stopping and restore the earliest best epoch; no hyperparameter "
    "search or post-run DEV retuning will be permitted."
)

In [ ]:
# STEP 6.30J
# FD004 SINGLE PROSPECTIVELY-FROZEN TRAINING RUN + EARLIEST-BEST RESTORE LOCK
#
# THIS IS THE ONE AUTHORIZED FD004 MODEL-TRAINING RUN.
# No hyperparameter search, scheduler, clipping, TEST access, official-RUL access,
# threshold selection, or post-run DEV retuning is permitted.

import copy
import hashlib
import json
import random
import time

import numpy as np
import torch


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630j(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30J hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_tensor_630j(value):
    tensor = value.detach().cpu().contiguous()
    return hashlib.sha256(
        tensor.numpy().tobytes(order="C")
    ).hexdigest()


def _sha_model_state_630j(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630j(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions and one-run gate
# =============================================================================

if not bool(globals().get("all_checks_passed_630i", False)):
    raise RuntimeError("STEP 6.30I must pass before STEP 6.30J.")

if bool(globals().get("FD004_TRAINING_RUN_COMPLETED_630J", False)):
    raise RuntimeError(
        "The single authorized FD004 training run has already completed."
    )

if bool(globals().get("FD004_TRAINING_RUN_IN_PROGRESS_630J", False)):
    raise RuntimeError(
        "FD004 training is marked in progress; do not launch another run."
    )

required_630j = [
    "MODEL_FD004_630I",
    "LOSS_FD004_630I",
    "OPTIMIZER_FD004_630I",
    "FD004_FIT_LOADER_630I",
    "FD004_DEV_LOADER_630I",
    "FD004_FIT_LOADER_GENERATOR_630I",
    "FD004_DEVICE_630I",
    "FD004_INITIAL_MODEL_SHA256_630I",
    "FD004_FIT_LOADER_GENERATOR_STATE_SHA256_630I",
    "FD004_INITIALIZATION_CONTRACT_SHA256_630I",
    "FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
    "FD004_TRAINING_SEED_630H",
    "FD004_MAX_EPOCHS_630H",
    "FD004_EARLY_STOPPING_PATIENCE_630H",
    "FD004_EARLY_STOPPING_MIN_DELTA_630H",
    "FD004_RESTORE_BEST_WEIGHTS_630H",
    "FD004_TIE_BREAK_RULE_630H",
    "FD004_LR_SCHEDULER_630H",
    "FD004_GRADIENT_CLIPPING_630H",
    "FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H",
    "X_FIT_FD004_630G",
    "y_FIT_FD004_630G",
    "X_DEV_FD004_630G",
    "y_DEV_FD004_630G",
]

missing_630j = [
    name_630j
    for name_630j in required_630j
    if name_630j not in globals()
]

if missing_630j:
    raise RuntimeError(
        "Missing required STEP 6.30J object(s): "
        + ", ".join(missing_630j)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630I", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30J.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630I", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30J.")

if FD004_LR_SCHEDULER_630H is not None:
    raise RuntimeError("Frozen FD004 protocol unexpectedly specifies a scheduler.")

if FD004_GRADIENT_CLIPPING_630H is not None:
    raise RuntimeError("Frozen FD004 protocol unexpectedly specifies clipping.")

if bool(FD004_DEV_BASED_RETUNING_AFTER_RUN_PERMITTED_630H):
    raise RuntimeError("Frozen FD004 protocol unexpectedly permits post-run retuning.")

if str(FD004_TIE_BREAK_RULE_630H) != "earliest epoch":
    raise RuntimeError("Unexpected FD004 early-stopping tie-break rule.")

if not bool(FD004_RESTORE_BEST_WEIGHTS_630H):
    raise RuntimeError("Frozen FD004 protocol does not require best-weight restore.")


# =============================================================================
# 3) Re-certify pristine initialization before first batch
# =============================================================================

model_sha_pretrain_630j = _sha_model_state_630j(
    MODEL_FD004_630I
)

if model_sha_pretrain_630j != FD004_INITIAL_MODEL_SHA256_630I:
    raise RuntimeError(
        "Persistent FD004 model no longer matches the frozen initial state."
    )

if len(OPTIMIZER_FD004_630I.state) != 0:
    raise RuntimeError(
        "Optimizer state is non-empty before the authorized training run."
    )

fit_generator_sha_pretrain_630j = _sha_tensor_630j(
    FD004_FIT_LOADER_GENERATOR_630I.get_state()
)

if (
    fit_generator_sha_pretrain_630j
    != FD004_FIT_LOADER_GENERATOR_STATE_SHA256_630I
):
    raise RuntimeError(
        "FIT DataLoader generator state changed before the training run."
    )

if len(FD004_FIT_LOADER_630I.dataset) != int(X_FIT_FD004_630G.shape[0]):
    raise RuntimeError("FIT DataLoader dataset size changed.")

if len(FD004_DEV_LOADER_630I.dataset) != int(X_DEV_FD004_630G.shape[0]):
    raise RuntimeError("DEV DataLoader dataset size changed.")


# =============================================================================
# 4) Transactional pre-run snapshot
# =============================================================================

model_state_snapshot_630j = copy.deepcopy(
    MODEL_FD004_630I.state_dict()
)

optimizer_state_snapshot_630j = copy.deepcopy(
    OPTIMIZER_FD004_630I.state_dict()
)

fit_generator_state_snapshot_630j = (
    FD004_FIT_LOADER_GENERATOR_630I.get_state().clone()
)

python_rng_snapshot_630j = random.getstate()
numpy_rng_snapshot_630j = np.random.get_state()
torch_cpu_rng_snapshot_630j = torch.get_rng_state().clone()

cuda_rng_snapshot_630j = (
    [state.clone() for state in torch.cuda.get_rng_state_all()]
    if torch.cuda.is_available()
    else None
)

model_training_mode_snapshot_630j = bool(
    MODEL_FD004_630I.training
)


# =============================================================================
# 5) Execute the one frozen training run
# =============================================================================

FD004_TRAINING_RUN_IN_PROGRESS_630J = True

history_630j = []
best_epoch_630j = None
best_dev_mae_630j = float("inf")
best_model_state_630j = None
best_model_sha_630j = None
best_dev_prediction_630j = None
epochs_without_improvement_630j = 0
stopped_early_630j = False

run_start_630j = time.perf_counter()

try:
    for epoch_630j in range(
        1,
        int(FD004_MAX_EPOCHS_630H) + 1,
    ):

        MODEL_FD004_630I.train()

        train_squared_error_sum_630j = 0.0
        train_example_count_630j = 0

        for xb_630j, yb_630j in FD004_FIT_LOADER_630I:

            xb_630j = xb_630j.to(
                FD004_DEVICE_630I,
                dtype=torch.float32,
                non_blocking=False,
            )

            yb_630j = yb_630j.to(
                FD004_DEVICE_630I,
                dtype=torch.float32,
                non_blocking=False,
            )

            OPTIMIZER_FD004_630I.zero_grad(
                set_to_none=True
            )

            pred_630j = MODEL_FD004_630I(
                xb_630j
            )

            loss_630j = LOSS_FD004_630I(
                pred_630j,
                yb_630j,
            )

            loss_630j.backward()

            # Frozen protocol: intentionally no gradient clipping.
            OPTIMIZER_FD004_630I.step()

            batch_n_630j = int(
                yb_630j.shape[0]
            )

            train_squared_error_sum_630j += (
                float(loss_630j.detach().item())
                * batch_n_630j
            )

            train_example_count_630j += batch_n_630j

        if train_example_count_630j != len(FD004_FIT_LOADER_630I.dataset):
            raise RuntimeError(
                "FIT epoch did not consume the complete frozen FIT population."
            )

        train_mse_630j = (
            train_squared_error_sum_630j
            / float(train_example_count_630j)
        )

        MODEL_FD004_630I.eval()

        dev_prediction_parts_630j = []
        dev_target_parts_630j = []

        with torch.no_grad():

            for xb_630j, yb_630j in FD004_DEV_LOADER_630I:

                xb_630j = xb_630j.to(
                    FD004_DEVICE_630I,
                    dtype=torch.float32,
                    non_blocking=False,
                )

                pred_630j = MODEL_FD004_630I(
                    xb_630j
                )

                dev_prediction_parts_630j.append(
                    pred_630j.detach().cpu().numpy().astype(
                        np.float32,
                        copy=False,
                    )
                )

                dev_target_parts_630j.append(
                    yb_630j.detach().cpu().numpy().astype(
                        np.float32,
                        copy=False,
                    )
                )

        dev_prediction_630j = np.concatenate(
            dev_prediction_parts_630j,
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )

        dev_target_check_630j = np.concatenate(
            dev_target_parts_630j,
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )

        if dev_prediction_630j.shape != y_DEV_FD004_630G.shape:
            raise RuntimeError("DEV prediction shape changed during monitoring.")

        if not np.array_equal(
            dev_target_check_630j,
            np.asarray(y_DEV_FD004_630G, dtype=np.float32),
        ):
            raise RuntimeError(
                "DEV DataLoader target order differs from the frozen DEV target order."
            )

        dev_mae_630j = float(
            np.mean(
                np.abs(
                    dev_prediction_630j.astype(np.float64)
                    -
                    np.asarray(
                        y_DEV_FD004_630G,
                        dtype=np.float64,
                    )
                ),
                dtype=np.float64,
            )
        )

        if not np.isfinite(train_mse_630j):
            raise RuntimeError("Non-finite FIT MSE encountered.")

        if not np.isfinite(dev_mae_630j):
            raise RuntimeError("Non-finite DEV MAE encountered.")

        improved_630j = bool(
            dev_mae_630j
            <
            (
                best_dev_mae_630j
                -
                float(FD004_EARLY_STOPPING_MIN_DELTA_630H)
            )
        )

        if improved_630j:

            best_dev_mae_630j = dev_mae_630j
            best_epoch_630j = int(epoch_630j)

            best_model_state_630j = copy.deepcopy(
                MODEL_FD004_630I.state_dict()
            )

            best_model_sha_630j = _sha_model_state_630j(
                MODEL_FD004_630I
            )

            best_dev_prediction_630j = np.array(
                dev_prediction_630j,
                dtype=np.float32,
                copy=True,
                order="C",
            )

            epochs_without_improvement_630j = 0

        else:

            epochs_without_improvement_630j += 1

        history_630j.append(
            {
                "epoch":
                    int(epoch_630j),
                "fit_mse":
                    float(train_mse_630j),
                "dev_mae":
                    float(dev_mae_630j),
                "improved":
                    bool(improved_630j),
                "epochs_without_improvement":
                    int(epochs_without_improvement_630j),
            }
        )

        print(
            f"Epoch {epoch_630j:03d} | "
            f"FIT MSE {train_mse_630j:.9f} | "
            f"DEV MAE {dev_mae_630j:.9f} | "
            f"best epoch {best_epoch_630j:03d} | "
            f"patience {epochs_without_improvement_630j}/"
            f"{int(FD004_EARLY_STOPPING_PATIENCE_630H)}"
        )

        if (
            epochs_without_improvement_630j
            >= int(FD004_EARLY_STOPPING_PATIENCE_630H)
        ):
            stopped_early_630j = True
            break

    if best_model_state_630j is None:
        raise RuntimeError("No finite best FD004 model state was captured.")

    if best_dev_prediction_630j is None:
        raise RuntimeError("No best-epoch DEV prediction vector was captured.")

    MODEL_FD004_630I.load_state_dict(
        best_model_state_630j,
        strict=True,
    )

    MODEL_FD004_630I.eval()

    restored_model_sha_630j = _sha_model_state_630j(
        MODEL_FD004_630I
    )

    if restored_model_sha_630j != best_model_sha_630j:
        raise RuntimeError(
            "Restored FD004 model does not match the captured earliest-best state."
        )

except BaseException:

    MODEL_FD004_630I.load_state_dict(
        model_state_snapshot_630j,
        strict=True,
    )

    OPTIMIZER_FD004_630I.load_state_dict(
        optimizer_state_snapshot_630j
    )

    FD004_FIT_LOADER_GENERATOR_630I.set_state(
        fit_generator_state_snapshot_630j
    )

    random.setstate(
        python_rng_snapshot_630j
    )

    np.random.set_state(
        numpy_rng_snapshot_630j
    )

    torch.set_rng_state(
        torch_cpu_rng_snapshot_630j
    )

    if (
        torch.cuda.is_available()
        and cuda_rng_snapshot_630j is not None
    ):
        torch.cuda.set_rng_state_all(
            cuda_rng_snapshot_630j
        )

    MODEL_FD004_630I.train(
        model_training_mode_snapshot_630j
    )

    FD004_TRAINING_RUN_IN_PROGRESS_630J = False

    raise


run_seconds_630j = float(
    time.perf_counter() - run_start_630j
)

epochs_executed_630j = int(
    len(history_630j)
)

if best_epoch_630j is None:
    raise RuntimeError("Best epoch was not resolved.")

if not (1 <= best_epoch_630j <= epochs_executed_630j):
    raise RuntimeError("Resolved best epoch is outside the executed epoch range.")

if epochs_executed_630j > int(FD004_MAX_EPOCHS_630H):
    raise RuntimeError("Training exceeded the frozen maximum epoch count.")


# =============================================================================
# 6) Freeze restored-best DEV prediction and history evidence
# =============================================================================

FD004_BEST_DEV_PREDICTION_630J = np.array(
    best_dev_prediction_630j,
    dtype=np.float32,
    copy=True,
    order="C",
)

FD004_BEST_DEV_PREDICTION_630J.setflags(
    write=False
)

FD004_TRAINING_HISTORY_630J = tuple(
    dict(row_630j)
    for row_630j in history_630j
)

FD004_BEST_EPOCH_630J = int(
    best_epoch_630j
)

FD004_BEST_DEV_MAE_630J = float(
    best_dev_mae_630j
)

FD004_EPOCHS_EXECUTED_630J = int(
    epochs_executed_630j
)

FD004_EARLY_STOPPED_630J = bool(
    stopped_early_630j
)

FD004_TRAINING_RUNTIME_SECONDS_630J = float(
    run_seconds_630j
)

FD004_TRAINED_MODEL_SHA256_630J = str(
    restored_model_sha_630j
)

FD004_BEST_DEV_PREDICTION_SHA256_630J = _sha_arr_630j(
    FD004_BEST_DEV_PREDICTION_630J
)

FD004_TRAINING_HISTORY_SHA256_630J = _sha_json_630j(
    list(FD004_TRAINING_HISTORY_630J)
)


# =============================================================================
# 7) Post-run safety checks
# =============================================================================

if FD004_TRAINED_MODEL_SHA256_630J == FD004_INITIAL_MODEL_SHA256_630I:
    raise RuntimeError(
        "Training completed but the restored model still equals the initial state."
    )

if len(OPTIMIZER_FD004_630I.state) == 0:
    raise RuntimeError(
        "Training completed but Adam state is unexpectedly empty."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Restored trained model must finish STEP 6.30J in evaluation mode."
    )


# =============================================================================
# 8) Freeze training-run evidence contract
# =============================================================================

FD004_TRAINING_RUN_CONTRACT_630J = {
    "step":
        "6.30J",
    "dataset":
        "FD004",
    "initialization_contract_sha256":
        FD004_INITIALIZATION_CONTRACT_SHA256_630I,
    "training_protocol_sha256":
        FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H,
    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,
    "single_authorized_run":
        True,
    "training_seed":
        int(FD004_TRAINING_SEED_630H),
    "monitor":
        "DEV_MAE",
    "min_delta":
        float(FD004_EARLY_STOPPING_MIN_DELTA_630H),
    "patience":
        int(FD004_EARLY_STOPPING_PATIENCE_630H),
    "maximum_epochs":
        int(FD004_MAX_EPOCHS_630H),
    "tie_break":
        "earliest epoch",
    "restore_best_weights":
        True,
    "lr_scheduler":
        None,
    "gradient_clipping":
        None,
    "hyperparameter_search":
        False,
    "post_run_dev_retuning":
        False,
    "epochs_executed":
        FD004_EPOCHS_EXECUTED_630J,
    "early_stopped":
        FD004_EARLY_STOPPED_630J,
    "best_epoch":
        FD004_BEST_EPOCH_630J,
    "best_dev_mae":
        FD004_BEST_DEV_MAE_630J,
    "initial_model_sha256":
        FD004_INITIAL_MODEL_SHA256_630I,
    "restored_best_model_sha256":
        FD004_TRAINED_MODEL_SHA256_630J,
    "best_dev_prediction_sha256":
        FD004_BEST_DEV_PREDICTION_SHA256_630J,
    "training_history_sha256":
        FD004_TRAINING_HISTORY_SHA256_630J,
    "test_contents_read":
        False,
    "official_rul_contents_read":
        False,
    "threshold_selected":
        False,
    "policy_selected":
        False,
}

FD004_TRAINING_RUN_CONTRACT_SHA256_630J = _sha_json_630j(
    FD004_TRAINING_RUN_CONTRACT_630J
)


# =============================================================================
# 9) Final run-state lock
# =============================================================================

FD004_TRAINING_RUN_IN_PROGRESS_630J = False
FD004_TRAINING_RUN_COMPLETED_630J = True
FD004_SINGLE_TRAINING_RUN_CONSUMED_630J = True

FD004_MODEL_TRAINING_PERFORMED_630J = True
FD004_DEV_MONITORING_PERFORMED_630J = True
FD004_POST_RUN_RETUNING_PERFORMED_630J = False

FD004_TEST_CONTENTS_READ_630J = False
FD004_RUL_CONTENTS_READ_630J = False
FD004_THRESHOLD_SELECTED_630J = False
FD004_POLICY_SELECTED_630J = False

all_checks_passed_630j = True


# =============================================================================
# 10) Report
# =============================================================================

print("\n" + "=" * 168)

print(
    "STEP 6.30J — FD004 SINGLE PROSPECTIVELY-FROZEN TRAINING RUN "
    "+ EARLIEST-BEST RESTORE LOCK"
)

print("=" * 168)

print("Single authorized run consumed :", True)
print("Model training performed       :", True)
print("DEV monitoring performed       :", True)
print("Hyperparameter search          :", False)
print("Post-run DEV retuning          :", False)
print("LR scheduler used              :", False)
print("Gradient clipping used         :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)

print("\n--- FROZEN TRAINING OUTCOME ---")

print(
    "Epochs executed                :",
    FD004_EPOCHS_EXECUTED_630J,
)

print(
    "Early stopped                  :",
    FD004_EARLY_STOPPED_630J,
)

print(
    "Best epoch                     :",
    FD004_BEST_EPOCH_630J,
)

print(
    "Best / restored DEV MAE        :",
    FD004_BEST_DEV_MAE_630J,
)

print(
    "Model mode after restore       :",
    "eval" if not MODEL_FD004_630I.training else "train",
)

print(
    "Optimizer state entries        :",
    len(OPTIMIZER_FD004_630I.state),
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Initial model SHA-256          :",
    FD004_INITIAL_MODEL_SHA256_630I,
)

print(
    "Restored trained SHA-256       :",
    FD004_TRAINED_MODEL_SHA256_630J,
)

print(
    "Best DEV prediction SHA-256    :",
    FD004_BEST_DEV_PREDICTION_SHA256_630J,
)

print(
    "Training history SHA-256       :",
    FD004_TRAINING_HISTORY_SHA256_630J,
)

print(
    "Training-run contract SHA-256  :",
    FD004_TRAINING_RUN_CONTRACT_SHA256_630J,
)

print(
    "\nPASS — the one prospectively frozen FD004 training run is complete. "
    "The earliest best DEV-MAE epoch was restored, and the trained model is now "
    "locked against retraining or post-run hyperparameter adjustment."
)

print(
    "Next: lock the deterministic clean-DEV baseline from the retained "
    "best-epoch DEV predictions, then perform the canonical clean-DEV "
    "MC-dropout uncertainty run. TEST and official RUL remain sealed."
)

In [ ]:
# STEP 6.30K
# FD004 DETERMINISTIC CLEAN-DEV BASELINE LOCK FROM RETAINED BEST-EPOCH PREDICTIONS
#
# NO NEW MODEL INFERENCE.
# NO MC-DROPOUT.
# NO THRESHOLD / POLICY SELECTION.
# NO TEST / OFFICIAL RUL ACCESS.

import hashlib
import json
import numpy as np


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630k(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30K hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_model_state_630k(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630k(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630j", False)):
    raise RuntimeError("STEP 6.30J must pass before STEP 6.30K.")

if not bool(globals().get("FD004_TRAINING_RUN_COMPLETED_630J", False)):
    raise RuntimeError("The authorized FD004 training run is not locked complete.")

if not bool(globals().get("FD004_SINGLE_TRAINING_RUN_CONSUMED_630J", False)):
    raise RuntimeError("The single FD004 training run was not marked consumed.")

if bool(globals().get("FD004_CLEAN_DEV_BASELINE_LOCKED_630K", False)):
    raise RuntimeError("STEP 6.30K has already been completed.")

required_630k = [
    "MODEL_FD004_630I",
    "FD004_BEST_DEV_PREDICTION_630J",
    "FD004_BEST_DEV_PREDICTION_SHA256_630J",
    "FD004_BEST_DEV_MAE_630J",
    "FD004_BEST_EPOCH_630J",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_TRAINING_RUN_CONTRACT_SHA256_630J",
    "y_DEV_FD004_630G",
    "FD004_Y_DEV_SHA256_630G",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
]

missing_630k = [
    name_630k
    for name_630k in required_630k
    if name_630k not in globals()
]

if missing_630k:
    raise RuntimeError(
        "Missing required STEP 6.30K object(s): "
        + ", ".join(missing_630k)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630J", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30K.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630J", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30K.")


# =============================================================================
# 3) Re-certify frozen trained model and retained prediction
# =============================================================================

model_sha_before_630k = _sha_model_state_630k(
    MODEL_FD004_630I
)

if model_sha_before_630k != FD004_TRAINED_MODEL_SHA256_630J:
    raise RuntimeError(
        "The restored trained FD004 model changed after STEP 6.30J."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "The restored trained FD004 model is not in evaluation mode."
    )

prediction_630k = np.asarray(
    FD004_BEST_DEV_PREDICTION_630J,
    dtype=np.float32,
)

target_630k = np.asarray(
    y_DEV_FD004_630G,
    dtype=np.float32,
)

if _sha_arr_630k(prediction_630k) != FD004_BEST_DEV_PREDICTION_SHA256_630J:
    raise RuntimeError(
        "Retained best-epoch DEV prediction fingerprint changed."
    )

if _sha_arr_630k(target_630k) != FD004_Y_DEV_SHA256_630G:
    raise RuntimeError(
        "Frozen DEV target fingerprint changed."
    )

if prediction_630k.shape != target_630k.shape:
    raise RuntimeError(
        "Retained DEV prediction and target shapes differ."
    )

if prediction_630k.ndim != 1:
    raise RuntimeError(
        "Retained clean DEV prediction must be one-dimensional."
    )

if prediction_630k.shape[0] != 10824:
    raise RuntimeError(
        "Unexpected FD004 DEV window population."
    )

if not np.all(np.isfinite(prediction_630k)):
    raise RuntimeError(
        "Retained clean DEV prediction contains non-finite values."
    )

if not np.all(np.isfinite(target_630k)):
    raise RuntimeError(
        "Frozen DEV target contains non-finite values."
    )


# =============================================================================
# 4) Lock deterministic clean-DEV baseline — no new inference
# =============================================================================

FD004_CLEAN_DEV_PREDICTION_630K = np.array(
    prediction_630k,
    dtype=np.float32,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_PREDICTION_630K.setflags(
    write=False
)

error_630k = (
    FD004_CLEAN_DEV_PREDICTION_630K.astype(np.float64)
    -
    target_630k.astype(np.float64)
)

abs_error_630k = np.abs(
    error_630k
)

FD004_CLEAN_DEV_ABS_ERROR_630K = np.array(
    abs_error_630k,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_ABS_ERROR_630K.setflags(
    write=False
)

FD004_CLEAN_DEV_MAE_630K = float(
    np.mean(
        abs_error_630k,
        dtype=np.float64,
    )
)

FD004_CLEAN_DEV_MEDIAN_ABS_ERROR_630K = float(
    np.median(
        abs_error_630k
    )
)

FD004_CLEAN_DEV_RMSE_630K = float(
    np.sqrt(
        np.mean(
            np.square(error_630k),
            dtype=np.float64,
        )
    )
)

monitor_abs_diff_630k = float(
    abs(
        FD004_CLEAN_DEV_MAE_630K
        -
        float(FD004_BEST_DEV_MAE_630J)
    )
)

monitor_consistency_630k = bool(
    monitor_abs_diff_630k <= 1e-12
)

if not monitor_consistency_630k:
    raise RuntimeError(
        "Clean DEV MAE does not exactly reproduce the frozen best-epoch monitor."
    )


# =============================================================================
# 5) Fingerprints
# =============================================================================

FD004_CLEAN_DEV_PREDICTION_SHA256_630K = _sha_arr_630k(
    FD004_CLEAN_DEV_PREDICTION_630K
)

FD004_CLEAN_DEV_ABS_ERROR_SHA256_630K = _sha_arr_630k(
    FD004_CLEAN_DEV_ABS_ERROR_630K
)

if (
    FD004_CLEAN_DEV_PREDICTION_SHA256_630K
    != FD004_BEST_DEV_PREDICTION_SHA256_630J
):
    raise RuntimeError(
        "Clean DEV baseline prediction does not preserve retained best-epoch bytes."
    )


# =============================================================================
# 6) Baseline contract
# =============================================================================

FD004_CLEAN_DEV_BASELINE_CONTRACT_630K = {
    "step":
        "6.30K",

    "dataset":
        "FD004",

    "source":
        "retained earliest-best DEV prediction from STEP 6.30J",

    "new_model_inference_performed":
        False,

    "mc_dropout_execution_performed":
        False,

    "best_epoch":
        int(FD004_BEST_EPOCH_630J),

    "population":
        int(FD004_CLEAN_DEV_PREDICTION_630K.shape[0]),

    "prediction_dtype":
        str(FD004_CLEAN_DEV_PREDICTION_630K.dtype),

    "target_dtype":
        str(target_630k.dtype),

    "clean_dev_mae":
        FD004_CLEAN_DEV_MAE_630K,

    "clean_dev_median_absolute_error":
        FD004_CLEAN_DEV_MEDIAN_ABS_ERROR_630K,

    "clean_dev_rmse":
        FD004_CLEAN_DEV_RMSE_630K,

    "training_monitor_dev_mae":
        float(FD004_BEST_DEV_MAE_630J),

    "monitor_absolute_difference":
        monitor_abs_diff_630k,

    "monitor_consistency_exact_tolerance_1e_12":
        monitor_consistency_630k,

    "trained_model_sha256":
        FD004_TRAINED_MODEL_SHA256_630J,

    "prediction_sha256":
        FD004_CLEAN_DEV_PREDICTION_SHA256_630K,

    "absolute_error_sha256":
        FD004_CLEAN_DEV_ABS_ERROR_SHA256_630K,

    "dev_target_sha256":
        FD004_Y_DEV_SHA256_630G,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "training_run_contract_sha256":
        FD004_TRAINING_RUN_CONTRACT_SHA256_630J,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_CLEAN_DEV_BASELINE_CONTRACT_SHA256_630K = _sha_json_630k(
    FD004_CLEAN_DEV_BASELINE_CONTRACT_630K
)


# =============================================================================
# 7) Final safety verification
# =============================================================================

model_sha_after_630k = _sha_model_state_630k(
    MODEL_FD004_630I
)

if model_sha_after_630k != model_sha_before_630k:
    raise RuntimeError(
        "Model state changed during deterministic baseline locking."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Model mode changed during deterministic baseline locking."
    )


# =============================================================================
# 8) State lock
# =============================================================================

FD004_CLEAN_DEV_BASELINE_LOCKED_630K = True

FD004_NEW_MODEL_INFERENCE_PERFORMED_630K = False
FD004_MC_DROPOUT_EXECUTION_PERFORMED_630K = False
FD004_MODEL_RETRAINED_630K = False
FD004_THRESHOLD_SELECTED_630K = False
FD004_POLICY_SELECTED_630K = False

FD004_TEST_CONTENTS_READ_630K = False
FD004_RUL_CONTENTS_READ_630K = False

all_checks_passed_630k = True


# =============================================================================
# 9) Report
# =============================================================================

print("=" * 164)

print(
    "STEP 6.30K — FD004 DETERMINISTIC CLEAN-DEV BASELINE "
    "LOCK FROM RETAINED BEST-EPOCH PREDICTIONS"
)

print("=" * 164)

print("New model inference performed :", False)
print("MC-dropout execution performed:", False)
print("Model retrained               :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- CLEAN DEV BASELINE ---")

print(
    "Population                    :",
    FD004_CLEAN_DEV_PREDICTION_630K.shape[0],
)

print(
    "Best epoch source             :",
    FD004_BEST_EPOCH_630J,
)

print(
    "Clean DEV MAE                 :",
    FD004_CLEAN_DEV_MAE_630K,
)

print(
    "Median absolute error         :",
    FD004_CLEAN_DEV_MEDIAN_ABS_ERROR_630K,
)

print(
    "Clean DEV RMSE                :",
    FD004_CLEAN_DEV_RMSE_630K,
)

print(
    "Frozen training-monitor MAE   :",
    FD004_BEST_DEV_MAE_630J,
)

print(
    "Monitor absolute difference   :",
    monitor_abs_diff_630k,
)

print(
    "Monitor consistency           :",
    monitor_consistency_630k,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Prediction SHA-256            :",
    FD004_CLEAN_DEV_PREDICTION_SHA256_630K,
)

print(
    "Absolute-error SHA-256        :",
    FD004_CLEAN_DEV_ABS_ERROR_SHA256_630K,
)

print(
    "Trained model SHA-256         :",
    FD004_TRAINED_MODEL_SHA256_630J,
)

print(
    "Baseline contract SHA-256     :",
    FD004_CLEAN_DEV_BASELINE_CONTRACT_SHA256_630K,
)

print(
    "\nPASS — the deterministic FD004 clean-DEV baseline is locked directly "
    "from the retained earliest-best prediction vector with no new inference."
)

print(
    "Next: prospectively freeze/verify the canonical FD004 clean-DEV "
    "MC-dropout protocol, then execute the single clean-DEV MC uncertainty run. "
    "TEST and official RUL remain sealed."
)

In [ ]:
# STEP 6.30L
# FD004 CANONICAL CLEAN-DEV MC-DROPOUT PROTOCOL PROSPECTIVE FREEZE
#
# FREEZE ONLY:
#   - no MC execution
#   - no new model inference
#   - no DEV outcome/error use
#   - no threshold/policy selection
#   - no TEST / official RUL access

import hashlib
import json


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_model_state_630l(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630l(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630k", False)):
    raise RuntimeError("STEP 6.30K must pass before STEP 6.30L.")

if not bool(globals().get("FD004_CLEAN_DEV_BASELINE_LOCKED_630K", False)):
    raise RuntimeError("FD004 clean DEV baseline is not locked.")

if bool(globals().get("FD004_MC_PROTOCOL_LOCKED_630L", False)):
    raise RuntimeError("STEP 6.30L has already been completed.")

required_630l = [
    "MODEL_FD004_630I",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_CLEAN_DEV_PREDICTION_SHA256_630K",
    "FD004_CLEAN_DEV_BASELINE_CONTRACT_SHA256_630K",
    "FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
    "FD004_TRAINING_SEED_630H",
    "FD004_EVAL_BATCH_SIZE_630H",
    "X_DEV_FD004_630G",
]

missing_630l = [
    name_630l
    for name_630l in required_630l
    if name_630l not in globals()
]

if missing_630l:
    raise RuntimeError(
        "Missing required STEP 6.30L object(s): "
        + ", ".join(missing_630l)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630K", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30L.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630K", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30L.")


# =============================================================================
# 3) Re-certify frozen trained model / clean DEV authority
# =============================================================================

model_sha_before_630l = _sha_model_state_630l(
    MODEL_FD004_630I
)

if model_sha_before_630l != FD004_TRAINED_MODEL_SHA256_630J:
    raise RuntimeError(
        "Frozen trained FD004 model changed before MC-protocol freeze."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Frozen trained FD004 model must enter STEP 6.30L in eval mode."
    )

if tuple(X_DEV_FD004_630G.shape) != (10824, 30, 24):
    raise RuntimeError(
        "Frozen FD004 DEV window tensor no longer has shape (10824, 30, 24)."
    )


# =============================================================================
# 4) Freeze canonical clean-DEV MC-dropout protocol
# =============================================================================

FD004_MC_PASSES_630L = 50
FD004_MC_BATCH_SIZE_630L = int(
    FD004_EVAL_BATCH_SIZE_630H
)
FD004_MC_SEED_630L = int(
    FD004_TRAINING_SEED_630H
)
FD004_MC_STD_DDOF_630L = 0

if FD004_MC_BATCH_SIZE_630L != 512:
    raise RuntimeError(
        "Frozen FD004 evaluation batch size is not the canonical MC batch size 512."
    )

if FD004_MC_SEED_630L != 42:
    raise RuntimeError(
        "Frozen FD004 seed is not the canonical MC seed 42."
    )

FD004_MC_PROTOCOL_METHOD_630L = {
    "passes":
        FD004_MC_PASSES_630L,

    "batch_size":
        FD004_MC_BATCH_SIZE_630L,

    "seed":
        FD004_MC_SEED_630L,

    "std_ddof":
        FD004_MC_STD_DDOF_630L,

    "input_population":
        "all frozen clean DEV windows in original frozen DEV-window order",

    "input_shape":
        [10824, 30, 24],

    "dropout_behavior":
        "enable dropout by temporary model.train() during MC forward passes",

    "gradient_behavior":
        "torch.no_grad() for all MC forward passes",

    "pass_order":
        "pass-major; each pass traverses DEV windows once in fixed non-shuffled order",

    "prediction_matrix_shape":
        [50, 10824],

    "prediction_matrix_dtype":
        "float32",

    "mc_mean_rule":
        "np.mean(all_predictions.astype(float64), axis=0)",

    "mc_std_rule":
        "np.std(all_predictions.astype(float64), axis=0, ddof=0)",

    "mc_mean_dtype":
        "float64",

    "mc_std_dtype":
        "float64",

    "rng_rule":
        "set torch CPU seed=42 and torch CUDA seed-all=42 immediately before MC execution",

    "model_mode_restore_rule":
        "restore the exact pre-MC training/eval mode after execution",

    "model_state_rule":
        "model parameter/state fingerprint must be identical before and after MC",

    "deterministic_prediction_role":
        "retain STEP 6.30K deterministic clean-DEV prediction separately; MC mean does not replace it",

    "dev_targets_or_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


# =============================================================================
# 5) Freeze prospective MC protocol contract
# =============================================================================

FD004_MC_PROTOCOL_CONTRACT_630L = {
    "step":
        "6.30L",

    "dataset":
        "FD004",

    "purpose":
        "prospective freeze of the canonical clean-DEV MC-dropout uncertainty execution before MC inference",

    "cross_dataset_methodology":
        "same canonical MC-dropout methodology established for prior datasets; no prior-dataset numerical uncertainty or calibration references inherited",

    "trained_model_sha256":
        FD004_TRAINED_MODEL_SHA256_630J,

    "clean_dev_prediction_sha256":
        FD004_CLEAN_DEV_PREDICTION_SHA256_630K,

    "clean_dev_baseline_contract_sha256":
        FD004_CLEAN_DEV_BASELINE_CONTRACT_SHA256_630K,

    "training_protocol_sha256":
        FD004_TRAINING_PROTOCOL_CONTRACT_SHA256_630H,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "method":
        dict(FD004_MC_PROTOCOL_METHOD_630L),

    "mc_execution_performed":
        False,

    "new_model_inference_performed":
        False,

    "dev_outcome_or_error_used":
        False,

    "model_retrained":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_MC_PROTOCOL_CONTRACT_SHA256_630L = _sha_json_630l(
    FD004_MC_PROTOCOL_CONTRACT_630L
)


# =============================================================================
# 6) Final safety verification
# =============================================================================

model_sha_after_630l = _sha_model_state_630l(
    MODEL_FD004_630I
)

if model_sha_after_630l != model_sha_before_630l:
    raise RuntimeError(
        "Model state changed during MC-protocol freeze."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Model mode changed during MC-protocol freeze."
    )


# =============================================================================
# 7) State lock
# =============================================================================

FD004_MC_PROTOCOL_LOCKED_630L = True
FD004_MC_EXECUTION_PERFORMED_630L = False
FD004_NEW_MODEL_INFERENCE_PERFORMED_630L = False
FD004_DEV_OUTCOMES_USED_630L = False
FD004_MODEL_RETRAINED_630L = False
FD004_THRESHOLD_SELECTED_630L = False
FD004_POLICY_SELECTED_630L = False

FD004_TEST_CONTENTS_READ_630L = False
FD004_RUL_CONTENTS_READ_630L = False

all_checks_passed_630l = True


# =============================================================================
# 8) Report
# =============================================================================

print("=" * 164)

print(
    "STEP 6.30L — FD004 CANONICAL CLEAN-DEV MC-DROPOUT "
    "PROTOCOL PROSPECTIVE FREEZE"
)

print("=" * 164)

print("MC execution performed        :", False)
print("New model inference performed :", False)
print("DEV outcomes/errors used      :", False)
print("Model retrained               :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- FROZEN CANONICAL MC PROTOCOL ---")

print("MC passes                     :", FD004_MC_PASSES_630L)
print("MC batch size                 :", FD004_MC_BATCH_SIZE_630L)
print("MC seed                       :", FD004_MC_SEED_630L)
print("MC STD ddof                   :", FD004_MC_STD_DDOF_630L)
print("DEV windows                   :", X_DEV_FD004_630G.shape[0])
print("Expected MC matrix shape      :", (50, 10824))
print("MC matrix dtype               :", "float32")
print("MC mean / std dtype           :", "float64 / float64")
print("Dropout enabled during MC     :", True)
print("Gradients enabled during MC   :", False)
print("DEV order shuffled            :", False)
print("Restore pre-MC model mode     :", True)
print("Require model-state identity  :", True)

print("\n--- AUTHORITY FINGERPRINTS ---")

print(
    "Trained model SHA-256         :",
    FD004_TRAINED_MODEL_SHA256_630J,
)

print(
    "Clean DEV prediction SHA-256  :",
    FD004_CLEAN_DEV_PREDICTION_SHA256_630K,
)

print(
    "MC protocol contract SHA-256  :",
    FD004_MC_PROTOCOL_CONTRACT_SHA256_630L,
)

print(
    "\nPASS — the canonical FD004 clean-DEV MC-dropout protocol is "
    "prospectively frozen before any MC execution."
)

print(
    "Next: execute exactly one canonical 50-pass clean-DEV MC-dropout run, "
    "retain the MC matrix/mean/std, verify model-state and mode restoration, "
    "and keep TEST and official RUL sealed."
)

In [ ]:
# STEP 6.30M
# FD004 SINGLE CANONICAL CLEAN-DEV MC-DROPOUT EXECUTION + UNCERTAINTY LOCK
#
# EXECUTION:
#   - exactly one 50-pass clean-DEV MC-dropout run
#   - no DEV targets/errors
#   - no threshold/policy selection
#   - no TEST / official RUL access
#   - model parameters must remain unchanged
#   - pre-MC model mode must be restored

import hashlib
import json
import time

import numpy as np
import torch


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_arr_630m(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30M hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_tensor_630m(value):
    tensor = value.detach().cpu().contiguous()
    return hashlib.sha256(
        tensor.numpy().tobytes(order="C")
    ).hexdigest()


def _sha_model_state_630m(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630m(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions and single-execution gate
# =============================================================================

if not bool(globals().get("all_checks_passed_630l", False)):
    raise RuntimeError("STEP 6.30L must pass before STEP 6.30M.")

if not bool(globals().get("FD004_MC_PROTOCOL_LOCKED_630L", False)):
    raise RuntimeError("FD004 MC protocol is not frozen.")

if bool(globals().get("FD004_MC_EXECUTION_COMPLETED_630M", False)):
    raise RuntimeError("STEP 6.30M canonical MC execution has already completed.")

required_630m = [
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_X_DEV_TENSOR_630I",
    "FD004_X_DEV_SHA256_630G",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_PREDICTION_SHA256_630K",
    "FD004_MC_PROTOCOL_CONTRACT_SHA256_630L",
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_STD_DDOF_630L",
]

missing_630m = [
    name_630m
    for name_630m in required_630m
    if name_630m not in globals()
]

if missing_630m:
    raise RuntimeError(
        "Missing required STEP 6.30M object(s): "
        + ", ".join(missing_630m)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630L", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30M.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630L", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30M.")

if int(FD004_MC_PASSES_630L) != 50:
    raise RuntimeError("Frozen MC pass count changed.")

if int(FD004_MC_BATCH_SIZE_630L) != 512:
    raise RuntimeError("Frozen MC batch size changed.")

if int(FD004_MC_SEED_630L) != 42:
    raise RuntimeError("Frozen MC seed changed.")

if int(FD004_MC_STD_DDOF_630L) != 0:
    raise RuntimeError("Frozen MC standard-deviation ddof changed.")


# =============================================================================
# 3) Re-certify model and clean DEV input authority
# =============================================================================

model_sha_before_630m = _sha_model_state_630m(
    MODEL_FD004_630I
)

if model_sha_before_630m != FD004_TRAINED_MODEL_SHA256_630J:
    raise RuntimeError(
        "Frozen trained FD004 model changed before MC execution."
    )

pre_mc_training_mode_630m = bool(
    MODEL_FD004_630I.training
)

if pre_mc_training_mode_630m:
    raise RuntimeError(
        "Canonical FD004 MC execution must begin from the frozen eval-mode model."
    )

if _sha_tensor_630m(FD004_X_DEV_TENSOR_630I) != FD004_X_DEV_SHA256_630G:
    raise RuntimeError(
        "Frozen FD004 DEV model-input tensor changed before MC execution."
    )

clean_prediction_630m = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float32,
)

if _sha_arr_630m(clean_prediction_630m) != FD004_CLEAN_DEV_PREDICTION_SHA256_630K:
    raise RuntimeError(
        "Frozen deterministic clean-DEV prediction changed before MC execution."
    )

n_dev_630m = int(
    FD004_X_DEV_TENSOR_630I.shape[0]
)

if n_dev_630m != 10824:
    raise RuntimeError("Unexpected FD004 clean DEV population.")

if tuple(FD004_X_DEV_TENSOR_630I.shape[1:]) != (30, 24):
    raise RuntimeError("Unexpected FD004 DEV window geometry.")


# =============================================================================
# 4) Execute exactly one canonical clean-DEV MC-dropout run
# =============================================================================

passes_630m = int(FD004_MC_PASSES_630L)
batch_size_630m = int(FD004_MC_BATCH_SIZE_630L)
seed_630m = int(FD004_MC_SEED_630L)

mc_matrix_630m = np.empty(
    (passes_630m, n_dev_630m),
    dtype=np.float32,
)

torch.manual_seed(seed_630m)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_630m)

run_start_630m = time.perf_counter()

try:
    MODEL_FD004_630I.train()

    with torch.no_grad():

        for pass_idx_630m in range(passes_630m):

            for start_630m in range(
                0,
                n_dev_630m,
                batch_size_630m,
            ):

                end_630m = min(
                    start_630m + batch_size_630m,
                    n_dev_630m,
                )

                xb_630m = FD004_X_DEV_TENSOR_630I[
                    start_630m:end_630m
                ].to(
                    FD004_DEVICE_630I,
                    dtype=torch.float32,
                    non_blocking=False,
                )

                pred_630m = MODEL_FD004_630I(
                    xb_630m
                )

                pred_np_630m = (
                    pred_630m
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(
                        np.float32,
                        copy=False,
                    )
                )

                if pred_np_630m.shape != (end_630m - start_630m,):
                    raise RuntimeError(
                        "Unexpected MC prediction batch shape."
                    )

                mc_matrix_630m[
                    pass_idx_630m,
                    start_630m:end_630m,
                ] = pred_np_630m

finally:
    MODEL_FD004_630I.train(
        pre_mc_training_mode_630m
    )

runtime_seconds_630m = float(
    time.perf_counter() - run_start_630m
)


# =============================================================================
# 5) Validate execution result
# =============================================================================

if mc_matrix_630m.shape != (50, 10824):
    raise RuntimeError(
        "Canonical FD004 MC prediction matrix has unexpected shape."
    )

if mc_matrix_630m.dtype != np.float32:
    raise RuntimeError(
        "Canonical FD004 MC prediction matrix is not float32."
    )

if not np.all(np.isfinite(mc_matrix_630m)):
    raise RuntimeError(
        "Canonical FD004 MC prediction matrix contains non-finite values."
    )

if MODEL_FD004_630I.training != pre_mc_training_mode_630m:
    raise RuntimeError(
        "FD004 model mode was not restored after MC execution."
    )

model_sha_after_630m = _sha_model_state_630m(
    MODEL_FD004_630I
)

if model_sha_after_630m != model_sha_before_630m:
    raise RuntimeError(
        "FD004 model state changed during MC-dropout execution."
    )


# =============================================================================
# 6) Compute canonical MC mean / population standard deviation
# =============================================================================

mc_matrix_float64_630m = mc_matrix_630m.astype(
    np.float64,
    copy=False,
)

mc_mean_630m = np.mean(
    mc_matrix_float64_630m,
    axis=0,
    dtype=np.float64,
)

mc_std_630m = np.std(
    mc_matrix_float64_630m,
    axis=0,
    ddof=int(FD004_MC_STD_DDOF_630L),
    dtype=np.float64,
)

if mc_mean_630m.shape != (10824,):
    raise RuntimeError("Unexpected FD004 MC-mean shape.")

if mc_std_630m.shape != (10824,):
    raise RuntimeError("Unexpected FD004 MC-STD shape.")

if mc_mean_630m.dtype != np.float64:
    raise RuntimeError("FD004 MC mean is not float64.")

if mc_std_630m.dtype != np.float64:
    raise RuntimeError("FD004 MC STD is not float64.")

if not np.all(np.isfinite(mc_mean_630m)):
    raise RuntimeError("FD004 MC mean contains non-finite values.")

if not np.all(np.isfinite(mc_std_630m)):
    raise RuntimeError("FD004 MC STD contains non-finite values.")

if np.any(mc_std_630m < 0.0):
    raise RuntimeError("FD004 MC STD contains negative values.")

if not np.any(mc_std_630m > 0.0):
    raise RuntimeError(
        "FD004 MC STD is identically zero; dropout uncertainty was not activated."
    )


# =============================================================================
# 7) Freeze MC evidence arrays
# =============================================================================

FD004_CLEAN_DEV_MC_MATRIX_630M = np.array(
    mc_matrix_630m,
    dtype=np.float32,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_MC_MEAN_630M = np.array(
    mc_mean_630m,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_MC_STD_630M = np.array(
    mc_std_630m,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_MC_MATRIX_630M.setflags(
    write=False
)

FD004_CLEAN_DEV_MC_MEAN_630M.setflags(
    write=False
)

FD004_CLEAN_DEV_MC_STD_630M.setflags(
    write=False
)

FD004_CLEAN_DEV_MC_RUNTIME_SECONDS_630M = runtime_seconds_630m


# =============================================================================
# 8) Label-free diagnostic against deterministic prediction
# =============================================================================

FD004_CLEAN_DEV_MC_MEAN_ABS_DIFF_FROM_DETERMINISTIC_630M = float(
    np.mean(
        np.abs(
            FD004_CLEAN_DEV_MC_MEAN_630M
            -
            clean_prediction_630m.astype(np.float64)
        ),
        dtype=np.float64,
    )
)


# =============================================================================
# 9) Evidence fingerprints
# =============================================================================

FD004_CLEAN_DEV_MC_MATRIX_SHA256_630M = _sha_arr_630m(
    FD004_CLEAN_DEV_MC_MATRIX_630M
)

FD004_CLEAN_DEV_MC_MEAN_SHA256_630M = _sha_arr_630m(
    FD004_CLEAN_DEV_MC_MEAN_630M
)

FD004_CLEAN_DEV_MC_STD_SHA256_630M = _sha_arr_630m(
    FD004_CLEAN_DEV_MC_STD_630M
)


# =============================================================================
# 10) MC execution evidence contract
# =============================================================================

FD004_MC_EXECUTION_CONTRACT_630M = {
    "step":
        "6.30M",

    "dataset":
        "FD004",

    "mc_protocol_contract_sha256":
        FD004_MC_PROTOCOL_CONTRACT_SHA256_630L,

    "trained_model_sha256":
        FD004_TRAINED_MODEL_SHA256_630J,

    "clean_dev_prediction_sha256":
        FD004_CLEAN_DEV_PREDICTION_SHA256_630K,

    "passes":
        passes_630m,

    "batch_size":
        batch_size_630m,

    "seed":
        seed_630m,

    "std_ddof":
        int(FD004_MC_STD_DDOF_630L),

    "population":
        n_dev_630m,

    "mc_matrix_shape":
        list(FD004_CLEAN_DEV_MC_MATRIX_630M.shape),

    "mc_matrix_dtype":
        str(FD004_CLEAN_DEV_MC_MATRIX_630M.dtype),

    "mc_mean_dtype":
        str(FD004_CLEAN_DEV_MC_MEAN_630M.dtype),

    "mc_std_dtype":
        str(FD004_CLEAN_DEV_MC_STD_630M.dtype),

    "mc_matrix_sha256":
        FD004_CLEAN_DEV_MC_MATRIX_SHA256_630M,

    "mc_mean_sha256":
        FD004_CLEAN_DEV_MC_MEAN_SHA256_630M,

    "mc_std_sha256":
        FD004_CLEAN_DEV_MC_STD_SHA256_630M,

    "model_state_unchanged":
        True,

    "model_mode_restored":
        True,

    "dev_targets_or_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_MC_EXECUTION_CONTRACT_SHA256_630M = _sha_json_630m(
    FD004_MC_EXECUTION_CONTRACT_630M
)


# =============================================================================
# 11) Final execution lock
# =============================================================================

FD004_MC_EXECUTION_COMPLETED_630M = True
FD004_MC_EXECUTION_PERFORMED_630M = True
FD004_NEW_MODEL_INFERENCE_PERFORMED_630M = True

FD004_DEV_OUTCOMES_USED_630M = False
FD004_MODEL_RETRAINED_630M = False
FD004_THRESHOLD_SELECTED_630M = False
FD004_POLICY_SELECTED_630M = False

FD004_TEST_CONTENTS_READ_630M = False
FD004_RUL_CONTENTS_READ_630M = False

all_checks_passed_630m = True


# =============================================================================
# 12) Report
# =============================================================================

std_summary_630m = (
    float(np.min(FD004_CLEAN_DEV_MC_STD_630M)),
    float(np.median(FD004_CLEAN_DEV_MC_STD_630M)),
    float(np.mean(FD004_CLEAN_DEV_MC_STD_630M, dtype=np.float64)),
    float(np.max(FD004_CLEAN_DEV_MC_STD_630M)),
)

print("=" * 164)

print(
    "STEP 6.30M — FD004 SINGLE CANONICAL CLEAN-DEV "
    "MC-DROPOUT EXECUTION + UNCERTAINTY LOCK"
)

print("=" * 164)

print("MC execution performed        :", True)
print("New model inference performed :", True)
print("DEV targets/errors used       :", False)
print("Model retrained               :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- CANONICAL MC EXECUTION ---")

print(
    "MC matrix shape / dtype       :",
    (
        FD004_CLEAN_DEV_MC_MATRIX_630M.shape,
        FD004_CLEAN_DEV_MC_MATRIX_630M.dtype,
    ),
)

print(
    "MC mean shape / dtype         :",
    (
        FD004_CLEAN_DEV_MC_MEAN_630M.shape,
        FD004_CLEAN_DEV_MC_MEAN_630M.dtype,
    ),
)

print(
    "MC STD shape / dtype          :",
    (
        FD004_CLEAN_DEV_MC_STD_630M.shape,
        FD004_CLEAN_DEV_MC_STD_630M.dtype,
    ),
)

print(
    "MC passes / batch / seed/ddof :",
    (
        passes_630m,
        batch_size_630m,
        seed_630m,
        int(FD004_MC_STD_DDOF_630L),
    ),
)

print(
    "MC runtime seconds            :",
    FD004_CLEAN_DEV_MC_RUNTIME_SECONDS_630M,
)

print(
    "MC STD min/median/mean/max    :",
    std_summary_630m,
)

print(
    "Mean |MC mean - deterministic|:",
    FD004_CLEAN_DEV_MC_MEAN_ABS_DIFF_FROM_DETERMINISTIC_630M,
)

print("\n--- MODEL SAFETY ---")

print(
    "Model state unchanged         :",
    model_sha_after_630m == model_sha_before_630m,
)

print(
    "Model mode restored           :",
    MODEL_FD004_630I.training == pre_mc_training_mode_630m,
)

print(
    "Final model mode              :",
    "eval" if not MODEL_FD004_630I.training else "train",
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "MC matrix SHA-256             :",
    FD004_CLEAN_DEV_MC_MATRIX_SHA256_630M,
)

print(
    "MC mean SHA-256               :",
    FD004_CLEAN_DEV_MC_MEAN_SHA256_630M,
)

print(
    "MC STD SHA-256                :",
    FD004_CLEAN_DEV_MC_STD_SHA256_630M,
)

print(
    "MC execution contract SHA-256 :",
    FD004_MC_EXECUTION_CONTRACT_SHA256_630M,
)

print(
    "\nPASS — exactly one canonical FD004 clean-DEV MC-dropout run is complete, "
    "fingerprinted, and locked with the trained model unchanged and eval mode restored."
)

print(
    "Next: construct the FD004 FIT-only phase/reference calibration required for "
    "the label-free reliability-risk chain before creating any DEV risk threshold. "
    "TEST and official RUL remain sealed."
)

In [ ]:
# STEP 6.30N
# FD004 LABEL-FREE RELIABILITY-CALIBRATION PROTOCOL PROSPECTIVE FREEZE
#
# FREEZE ONLY.
# NO FIT inference.
# NO FIT MC execution.
# NO DEV outcomes/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.

import hashlib
import json


# =============================================================================
# 1) Canonical helpers
# =============================================================================

def _sha_model_state_630n(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()

        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630n(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 2) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630m", False)):
    raise RuntimeError("STEP 6.30M must pass before STEP 6.30N.")

if not bool(globals().get("FD004_MC_EXECUTION_COMPLETED_630M", False)):
    raise RuntimeError(
        "The canonical FD004 clean-DEV MC run is not locked complete."
    )

if bool(
    globals().get(
        "FD004_RELIABILITY_CALIBRATION_PROTOCOL_LOCKED_630N",
        False,
    )
):
    raise RuntimeError("STEP 6.30N has already been completed.")

required_630n = [
    "MODEL_FD004_630I",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_CLEAN_DEV_PREDICTION_SHA256_630K",
    "FD004_CLEAN_DEV_MC_STD_SHA256_630M",
    "FD004_MC_EXECUTION_CONTRACT_SHA256_630M",
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_STD_DDOF_630L",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
]

missing_630n = [
    name
    for name in required_630n
    if name not in globals()
]

if missing_630n:
    raise RuntimeError(
        "Missing required STEP 6.30N object(s): "
        + ", ".join(missing_630n)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630M", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30N."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630M", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30N."
    )


# =============================================================================
# 3) Re-certify trained-model authority
# =============================================================================

model_sha_before_630n = _sha_model_state_630n(
    MODEL_FD004_630I
)

if model_sha_before_630n != FD004_TRAINED_MODEL_SHA256_630J:
    raise RuntimeError(
        "Frozen trained FD004 model changed before calibration-protocol freeze."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Frozen trained FD004 model must remain in eval mode."
    )


# =============================================================================
# 4) Freeze FD004 reliability calibration methodology
# =============================================================================

FD004_RELIABILITY_PHASE_COUNT_630N = 5

FD004_RELIABILITY_PHASE_QUANTILES_630N = (
    0.2,
    0.4,
    0.6,
    0.8,
)

FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N = "linear"

FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N = True


FD004_RELIABILITY_CALIBRATION_PROTOCOL_630N = {

    "step":
        "6.30N",

    "dataset":
        "FD004",

    "purpose":
        (
            "prospective freeze of the label-free reliability calibration "
            "chain before FIT deterministic/MC calibration inference"
        ),

    "cross_dataset_methodology":
        (
            "carry forward the established reliability methodology only; "
            "inherit no FD001/FD002/FD003 numerical phase edges, uncertainty "
            "references, G1/G2 references, threshold, or held-out result"
        ),

    # -------------------------------------------------------------------------
    # Base uncertainty
    # -------------------------------------------------------------------------

    "base_uncertainty": {

        "quantity":
            "MC-dropout predictive standard deviation",

        "passes":
            int(FD004_MC_PASSES_630L),

        "batch_size":
            int(FD004_MC_BATCH_SIZE_630L),

        "seed":
            int(FD004_MC_SEED_630L),

        "ddof":
            int(FD004_MC_STD_DDOF_630L),
    },

    # -------------------------------------------------------------------------
    # Prediction-phase definition
    # -------------------------------------------------------------------------

    "phase_definition": {

        "source_population":
            "FD004 FIT windows only",

        "phase_variable":
            "deterministic trained-model prediction",

        "phase_count":
            int(FD004_RELIABILITY_PHASE_COUNT_630N),

        "edge_quantiles":
            list(FD004_RELIABILITY_PHASE_QUANTILES_630N),

        "quantile_method":
            FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N,

        "assignment":
            "np.digitize(prediction, phase_edges, right=True)",

        "right":
            bool(FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N),
    },

    # -------------------------------------------------------------------------
    # FIT-only within-phase uncertainty calibration
    # -------------------------------------------------------------------------

    "phase_uncertainty_reference": {

        "source_population":
            "FD004 FIT windows only",

        "reference_quantity":
            "MC STD within each deterministic-prediction phase",

        "reference_sort":
            "ascending",

        "phase_adjustment":
            (
                "empirical midrank percentile within corresponding FIT phase: "
                "for exact ties use "
                "(left_index + right_index)/(2*n); "
                "otherwise use left_index/n; values below/above the "
                "reference naturally map to 0/1 via searchsorted"
            ),

        "labels_or_errors_used":
            False,
    },

    # -------------------------------------------------------------------------
    # G1 clean-DEV label-free ranking
    # -------------------------------------------------------------------------

    "g1": {

        "reference_population":
            "clean FD004 DEV windows only",

        "reference_quantity":
            "phase-adjusted uncertainty",

        "construction":
            "sorted clean-DEV phase-adjusted uncertainty",

        "mapping":
            (
                "right empirical CDF: "
                "searchsorted(reference, x, side='right') / n"
            ),

        "labels_or_errors_used":
            False,
    },

    # -------------------------------------------------------------------------
    # G2 engine-balanced clean-DEV ranking
    # -------------------------------------------------------------------------

    "g2": {

        "reference_population":
            "clean FD004 DEV windows only",

        "input_quantity":
            "G1 risk",

        "engine_weighting":
            (
                "equal total mass per DEV engine; each window weight is "
                "1/(number_of_DEV_engines * windows_for_that_engine)"
            ),

        "pairing":
            "pair G1 value with its window's equal-engine weight",

        "sort":
            (
                "stable ascending G1 using "
                "np.argsort(..., kind='mergesort')"
            ),

        "ties":
            (
                "aggregate equal G1 values and assign weighted mid-mass"
            ),

        "mapping":
            (
                "exact reference tie -> weighted mid-mass; "
                "below minimum -> 0; above maximum -> 1; "
                "otherwise cumulative reference weight strictly below x"
            ),

        "labels_or_errors_used":
            False,
    },

    "final_clean_dev_selective_risk":
        "G2(G1(phase_adjusted_MC_STD))",

    # -------------------------------------------------------------------------
    # Explicit no-selection/no-leakage state
    # -------------------------------------------------------------------------

    "threshold_selection_performed_here":
        False,

    "policy_selection_performed_here":
        False,

    "dev_targets_or_errors_used_here":
        False,

    "corruption_outcomes_used_here":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    # -------------------------------------------------------------------------
    # Frozen authorities
    # -------------------------------------------------------------------------

    "authorities": {

        "trained_model_sha256":
            FD004_TRAINED_MODEL_SHA256_630J,

        "clean_dev_prediction_sha256":
            FD004_CLEAN_DEV_PREDICTION_SHA256_630K,

        "clean_dev_mc_std_sha256":
            FD004_CLEAN_DEV_MC_STD_SHA256_630M,

        "clean_dev_mc_execution_contract_sha256":
            FD004_MC_EXECUTION_CONTRACT_SHA256_630M,

        "window_contract_sha256":
            FD004_WINDOW_CONTRACT_SHA256_630G,
    },
}


# =============================================================================
# 5) Fingerprint prospective calibration protocol
# =============================================================================

FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N = (
    _sha_json_630n(
        FD004_RELIABILITY_CALIBRATION_PROTOCOL_630N
    )
)


# =============================================================================
# 6) Final model-safety verification
# =============================================================================

model_sha_after_630n = _sha_model_state_630n(
    MODEL_FD004_630I
)

if model_sha_after_630n != model_sha_before_630n:
    raise RuntimeError(
        "Model state changed during reliability-calibration protocol freeze."
    )

if MODEL_FD004_630I.training:
    raise RuntimeError(
        "Model mode changed during reliability-calibration protocol freeze."
    )


# =============================================================================
# 7) State lock
# =============================================================================

FD004_RELIABILITY_CALIBRATION_PROTOCOL_LOCKED_630N = True

FD004_FIT_CALIBRATION_INFERENCE_PERFORMED_630N = False
FD004_FIT_MC_CALIBRATION_PERFORMED_630N = False

FD004_DEV_OUTCOMES_USED_630N = False

FD004_THRESHOLD_SELECTED_630N = False
FD004_POLICY_SELECTED_630N = False

FD004_TEST_CONTENTS_READ_630N = False
FD004_RUL_CONTENTS_READ_630N = False

all_checks_passed_630n = True


# =============================================================================
# 8) Report
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30N — FD004 LABEL-FREE RELIABILITY-CALIBRATION "
    "PROTOCOL PROSPECTIVE FREEZE"
)

print("=" * 168)

print("FIT calibration inference run :", False)
print("FIT MC calibration run        :", False)
print("DEV outcomes/errors used      :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- FROZEN PHASE CALIBRATION ---")

print(
    "Phase count                   :",
    FD004_RELIABILITY_PHASE_COUNT_630N,
)

print(
    "Phase quantiles               :",
    FD004_RELIABILITY_PHASE_QUANTILES_630N,
)

print(
    "Quantile method               :",
    FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N,
)

print(
    "Phase assignment right=True   :",
    FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N,
)

print(
    "Phase source                  :",
    "FIT deterministic prediction only",
)

print(
    "Uncertainty source            :",
    "FIT MC STD only",
)

print(
    "Within-phase transform        :",
    "empirical midrank percentile",
)

print("\n--- FROZEN LABEL-FREE RISK CHAIN ---")

print(
    "G1 reference                  :",
    "clean DEV phase-adjusted uncertainty",
)

print(
    "G1 mapping                    :",
    "right empirical CDF",
)

print(
    "G2 weighting                  :",
    "equal total mass per DEV engine",
)

print(
    "G2 sorting                    :",
    "stable mergesort with paired weights",
)

print(
    "G2 tie handling               :",
    "weighted mid-mass",
)

print(
    "Final selective risk          :",
    "G2(G1(phase-adjusted MC STD))",
)

print(
    "Prior-dataset numeric refs    :",
    False,
)

print("\n--- AUTHORITY FINGERPRINTS ---")

print(
    "Trained model SHA-256         :",
    FD004_TRAINED_MODEL_SHA256_630J,
)

print(
    "Clean DEV MC STD SHA-256      :",
    FD004_CLEAN_DEV_MC_STD_SHA256_630M,
)

print(
    "Calibration protocol SHA-256  :",
    FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N,
)

print(
    "\nPASS — the FD004 label-free phase→G1→G2 reliability-calibration "
    "method is prospectively frozen before FIT-side calibration inference."
)

print(
    "Next: run one deterministic FIT inference and one canonical 50-pass "
    "FIT MC-dropout calibration run, then derive FD004-only phase edges "
    "and within-phase FIT uncertainty references. No DEV labels/errors, "
    "threshold, TEST, or official RUL will be used."
)

In [ ]:
# STEP 6.30O
# FD004 FIT-ONLY DETERMINISTIC + CANONICAL MC CALIBRATION EXECUTION
# + PHASE-EDGE / WITHIN-PHASE UNCERTAINTY-REFERENCE LOCK

import hashlib
import json
import time

import numpy as np
import torch


def _sha_arr_630o(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30O hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_tensor_630o(value):
    tensor = value.detach().cpu().contiguous()
    return hashlib.sha256(
        tensor.numpy().tobytes(order="C")
    ).hexdigest()


def _sha_model_state_630o(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(
            json.dumps(
                list(tensor.shape),
                separators=(",", ":"),
            ).encode("utf-8")
        )
        h.update(tensor.numpy().tobytes(order="C"))

    return h.hexdigest()


def _sha_json_630o(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630n", False)):
    raise RuntimeError("STEP 6.30N must pass before STEP 6.30O.")

if not bool(
    globals().get(
        "FD004_RELIABILITY_CALIBRATION_PROTOCOL_LOCKED_630N",
        False,
    )
):
    raise RuntimeError(
        "FD004 reliability-calibration protocol is not frozen."
    )

if bool(
    globals().get(
        "FD004_FIT_CALIBRATION_EXECUTION_COMPLETED_630O",
        False,
    )
):
    raise RuntimeError("STEP 6.30O has already been completed.")

required_630o = [
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_X_FIT_TENSOR_630I",
    "FD004_X_FIT_SHA256_630G",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N",
    "FD004_RELIABILITY_PHASE_COUNT_630N",
    "FD004_RELIABILITY_PHASE_QUANTILES_630N",
    "FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N",
    "FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N",
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_STD_DDOF_630L",
    "FD004_WINDOW_CONTRACT_SHA256_630G",
]

missing_630o = [
    name
    for name in required_630o
    if name not in globals()
]

if missing_630o:
    raise RuntimeError(
        "Missing required STEP 6.30O object(s): "
        + ", ".join(missing_630o)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630N", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30O.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630N", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30O.")

if int(FD004_RELIABILITY_PHASE_COUNT_630N) != 5:
    raise RuntimeError("Frozen phase count changed.")

if tuple(FD004_RELIABILITY_PHASE_QUANTILES_630N) != (
    0.2, 0.4, 0.6, 0.8
):
    raise RuntimeError("Frozen phase quantiles changed.")

if str(FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N) != "linear":
    raise RuntimeError("Frozen phase-quantile method changed.")

if not bool(FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N):
    raise RuntimeError("Frozen phase assignment no longer uses right=True.")

if (
    int(FD004_MC_PASSES_630L),
    int(FD004_MC_BATCH_SIZE_630L),
    int(FD004_MC_SEED_630L),
    int(FD004_MC_STD_DDOF_630L),
) != (50, 512, 42, 0):
    raise RuntimeError("Frozen canonical MC parameters changed.")


# =============================================================================
# 2) Re-certify frozen model + FIT-window authority
# =============================================================================

model_sha_before_630o = _sha_model_state_630o(
    MODEL_FD004_630I
)

if model_sha_before_630o != FD004_TRAINED_MODEL_SHA256_630J:
    raise RuntimeError(
        "Frozen trained FD004 model changed before FIT calibration."
    )

pre_execution_training_mode_630o = bool(
    MODEL_FD004_630I.training
)

if pre_execution_training_mode_630o:
    raise RuntimeError(
        "FD004 FIT calibration must begin from the frozen eval-mode model."
    )

if _sha_tensor_630o(FD004_X_FIT_TENSOR_630I) != FD004_X_FIT_SHA256_630G:
    raise RuntimeError(
        "Frozen FD004 FIT model-input tensor changed before calibration."
    )

n_fit_630o = int(FD004_X_FIT_TENSOR_630I.shape[0])

if n_fit_630o != 43204:
    raise RuntimeError("Unexpected FD004 FIT-window population.")

if tuple(FD004_X_FIT_TENSOR_630I.shape[1:]) != (30, 24):
    raise RuntimeError("Unexpected FD004 FIT-window geometry.")


# =============================================================================
# 3) Deterministic FIT inference
# =============================================================================

batch_size_630o = int(FD004_MC_BATCH_SIZE_630L)

fit_prediction_630o = np.empty(
    n_fit_630o,
    dtype=np.float32,
)

det_start_630o = time.perf_counter()

MODEL_FD004_630I.eval()

with torch.no_grad():

    for start_630o in range(
        0,
        n_fit_630o,
        batch_size_630o,
    ):

        end_630o = min(
            start_630o + batch_size_630o,
            n_fit_630o,
        )

        xb_630o = FD004_X_FIT_TENSOR_630I[
            start_630o:end_630o
        ].to(
            FD004_DEVICE_630I,
            dtype=torch.float32,
            non_blocking=False,
        )

        pred_630o = MODEL_FD004_630I(xb_630o)

        pred_np_630o = (
            pred_630o
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
        )

        if pred_np_630o.shape != (end_630o - start_630o,):
            raise RuntimeError(
                "Unexpected deterministic FIT prediction batch shape."
            )

        fit_prediction_630o[start_630o:end_630o] = pred_np_630o

det_runtime_630o = float(
    time.perf_counter() - det_start_630o
)

if not np.all(np.isfinite(fit_prediction_630o)):
    raise RuntimeError(
        "Deterministic FIT predictions contain non-finite values."
    )


# =============================================================================
# 4) One canonical FIT MC-dropout calibration run
# =============================================================================

passes_630o = int(FD004_MC_PASSES_630L)
seed_630o = int(FD004_MC_SEED_630L)
ddof_630o = int(FD004_MC_STD_DDOF_630L)

fit_mc_matrix_630o = np.empty(
    (passes_630o, n_fit_630o),
    dtype=np.float32,
)

torch.manual_seed(seed_630o)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_630o)

mc_start_630o = time.perf_counter()

try:
    MODEL_FD004_630I.train()

    with torch.no_grad():

        for pass_idx_630o in range(passes_630o):

            for start_630o in range(
                0,
                n_fit_630o,
                batch_size_630o,
            ):

                end_630o = min(
                    start_630o + batch_size_630o,
                    n_fit_630o,
                )

                xb_630o = FD004_X_FIT_TENSOR_630I[
                    start_630o:end_630o
                ].to(
                    FD004_DEVICE_630I,
                    dtype=torch.float32,
                    non_blocking=False,
                )

                pred_630o = MODEL_FD004_630I(xb_630o)

                pred_np_630o = (
                    pred_630o
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32, copy=False)
                )

                if pred_np_630o.shape != (end_630o - start_630o,):
                    raise RuntimeError(
                        "Unexpected FIT MC prediction batch shape."
                    )

                fit_mc_matrix_630o[
                    pass_idx_630o,
                    start_630o:end_630o,
                ] = pred_np_630o

finally:
    MODEL_FD004_630I.train(
        pre_execution_training_mode_630o
    )

mc_runtime_630o = float(
    time.perf_counter() - mc_start_630o
)

if fit_mc_matrix_630o.shape != (50, 43204):
    raise RuntimeError("FIT MC matrix has unexpected shape.")

if fit_mc_matrix_630o.dtype != np.float32:
    raise RuntimeError("FIT MC matrix is not float32.")

if not np.all(np.isfinite(fit_mc_matrix_630o)):
    raise RuntimeError("FIT MC matrix contains non-finite values.")

if MODEL_FD004_630I.training != pre_execution_training_mode_630o:
    raise RuntimeError(
        "FD004 model mode was not restored after FIT MC execution."
    )

model_sha_after_mc_630o = _sha_model_state_630o(
    MODEL_FD004_630I
)

if model_sha_after_mc_630o != model_sha_before_630o:
    raise RuntimeError(
        "FD004 model state changed during FIT calibration inference."
    )


# =============================================================================
# 5) Canonical FIT MC mean / STD
# =============================================================================

fit_mc_float64_630o = fit_mc_matrix_630o.astype(
    np.float64,
    copy=False,
)

fit_mc_mean_630o = np.mean(
    fit_mc_float64_630o,
    axis=0,
    dtype=np.float64,
)

fit_mc_std_630o = np.std(
    fit_mc_float64_630o,
    axis=0,
    ddof=ddof_630o,
    dtype=np.float64,
)

if fit_mc_mean_630o.shape != (43204,):
    raise RuntimeError("FIT MC mean has unexpected shape.")

if fit_mc_std_630o.shape != (43204,):
    raise RuntimeError("FIT MC STD has unexpected shape.")

if not np.all(np.isfinite(fit_mc_mean_630o)):
    raise RuntimeError("FIT MC mean contains non-finite values.")

if not np.all(np.isfinite(fit_mc_std_630o)):
    raise RuntimeError("FIT MC STD contains non-finite values.")

if np.any(fit_mc_std_630o < 0.0):
    raise RuntimeError("FIT MC STD contains negative values.")

if not np.any(fit_mc_std_630o > 0.0):
    raise RuntimeError("FIT MC STD is identically zero.")


# =============================================================================
# 6) FD004-only phase edges
# =============================================================================

phase_quantiles_630o = np.asarray(
    FD004_RELIABILITY_PHASE_QUANTILES_630N,
    dtype=np.float64,
)

phase_edges_630o = np.quantile(
    fit_prediction_630o.astype(np.float64, copy=False),
    q=phase_quantiles_630o,
    method=FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N,
).astype(
    np.float64,
    copy=False,
)

if phase_edges_630o.shape != (4,):
    raise RuntimeError(
        "FD004 phase-edge vector has unexpected shape."
    )

if not np.all(np.isfinite(phase_edges_630o)):
    raise RuntimeError(
        "FD004 phase edges contain non-finite values."
    )

if np.any(np.diff(phase_edges_630o) < 0.0):
    raise RuntimeError(
        "FD004 phase edges are not nondecreasing."
    )

fit_phase_ids_630o = np.digitize(
    fit_prediction_630o.astype(np.float64, copy=False),
    phase_edges_630o,
    right=True,
).astype(
    np.int64,
    copy=False,
)

if np.any(
    (fit_phase_ids_630o < 0)
    |
    (
        fit_phase_ids_630o
        >= int(FD004_RELIABILITY_PHASE_COUNT_630N)
    )
):
    raise RuntimeError(
        "Invalid FD004 FIT phase IDs were created."
    )

phase_counts_630o = np.bincount(
    fit_phase_ids_630o,
    minlength=int(FD004_RELIABILITY_PHASE_COUNT_630N),
).astype(
    np.int64,
    copy=False,
)

if int(np.sum(phase_counts_630o)) != n_fit_630o:
    raise RuntimeError(
        "FD004 FIT phase counts do not sum to the FIT population."
    )

if np.any(phase_counts_630o <= 0):
    raise RuntimeError(
        "At least one FD004 FIT prediction phase is empty."
    )


# =============================================================================
# 7) FIT-only within-phase MC-STD references
# =============================================================================

phase_reference_arrays_630o = []

for phase_id_630o in range(
    int(FD004_RELIABILITY_PHASE_COUNT_630N)
):

    phase_mask_630o = (
        fit_phase_ids_630o
        == phase_id_630o
    )

    phase_reference_630o = np.sort(
        fit_mc_std_630o[
            phase_mask_630o
        ].astype(
            np.float64,
            copy=True,
        ),
        kind="mergesort",
    )

    if phase_reference_630o.shape[0] != int(
        phase_counts_630o[phase_id_630o]
    ):
        raise RuntimeError(
            f"Phase {phase_id_630o} reference size mismatch."
        )

    if not np.all(np.isfinite(phase_reference_630o)):
        raise RuntimeError(
            f"Phase {phase_id_630o} reference contains non-finite values."
        )

    if np.any(np.diff(phase_reference_630o) < 0.0):
        raise RuntimeError(
            f"Phase {phase_id_630o} reference is not sorted."
        )

    phase_reference_630o.setflags(write=False)

    phase_reference_arrays_630o.append(
        phase_reference_630o
    )


# =============================================================================
# 8) Freeze FIT calibration artifacts
# =============================================================================

FD004_FIT_DETERMINISTIC_PREDICTION_630O = np.array(
    fit_prediction_630o,
    dtype=np.float32,
    copy=True,
    order="C",
)

FD004_FIT_MC_MATRIX_630O = np.array(
    fit_mc_matrix_630o,
    dtype=np.float32,
    copy=True,
    order="C",
)

FD004_FIT_MC_MEAN_630O = np.array(
    fit_mc_mean_630o,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_FIT_MC_STD_630O = np.array(
    fit_mc_std_630o,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_PHASE_EDGES_630O = np.array(
    phase_edges_630o,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_FIT_PHASE_IDS_630O = np.array(
    fit_phase_ids_630o,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_FIT_PHASE_COUNTS_630O = np.array(
    phase_counts_630o,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_PHASE_MC_STD_REFERENCES_630O = tuple(
    phase_reference_arrays_630o
)

for array_630o in (
    FD004_FIT_DETERMINISTIC_PREDICTION_630O,
    FD004_FIT_MC_MATRIX_630O,
    FD004_FIT_MC_MEAN_630O,
    FD004_FIT_MC_STD_630O,
    FD004_PHASE_EDGES_630O,
    FD004_FIT_PHASE_IDS_630O,
    FD004_FIT_PHASE_COUNTS_630O,
):
    array_630o.setflags(write=False)

FD004_FIT_DETERMINISTIC_RUNTIME_SECONDS_630O = float(
    det_runtime_630o
)

FD004_FIT_MC_RUNTIME_SECONDS_630O = float(
    mc_runtime_630o
)


# =============================================================================
# 9) Fingerprints
# =============================================================================

FD004_FIT_DETERMINISTIC_PREDICTION_SHA256_630O = _sha_arr_630o(
    FD004_FIT_DETERMINISTIC_PREDICTION_630O
)

FD004_FIT_MC_MATRIX_SHA256_630O = _sha_arr_630o(
    FD004_FIT_MC_MATRIX_630O
)

FD004_FIT_MC_MEAN_SHA256_630O = _sha_arr_630o(
    FD004_FIT_MC_MEAN_630O
)

FD004_FIT_MC_STD_SHA256_630O = _sha_arr_630o(
    FD004_FIT_MC_STD_630O
)

FD004_PHASE_EDGES_SHA256_630O = _sha_arr_630o(
    FD004_PHASE_EDGES_630O
)

FD004_FIT_PHASE_IDS_SHA256_630O = _sha_arr_630o(
    FD004_FIT_PHASE_IDS_630O
)

FD004_PHASE_MC_STD_REFERENCE_SHA256_630O = tuple(
    _sha_arr_630o(ref_630o)
    for ref_630o in FD004_PHASE_MC_STD_REFERENCES_630O
)


# =============================================================================
# 10) Calibration evidence contract
# =============================================================================

FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_630O = {
    "step":
        "6.30O",

    "dataset":
        "FD004",

    "reliability_calibration_protocol_sha256":
        FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N,

    "window_contract_sha256":
        FD004_WINDOW_CONTRACT_SHA256_630G,

    "trained_model_sha256":
        FD004_TRAINED_MODEL_SHA256_630J,

    "fit_population":
        n_fit_630o,

    "deterministic_prediction_sha256":
        FD004_FIT_DETERMINISTIC_PREDICTION_SHA256_630O,

    "mc_matrix_sha256":
        FD004_FIT_MC_MATRIX_SHA256_630O,

    "mc_mean_sha256":
        FD004_FIT_MC_MEAN_SHA256_630O,

    "mc_std_sha256":
        FD004_FIT_MC_STD_SHA256_630O,

    "mc_passes":
        passes_630o,

    "mc_batch_size":
        batch_size_630o,

    "mc_seed":
        seed_630o,

    "mc_std_ddof":
        ddof_630o,

    "phase_count":
        int(FD004_RELIABILITY_PHASE_COUNT_630N),

    "phase_quantiles":
        list(FD004_RELIABILITY_PHASE_QUANTILES_630N),

    "phase_quantile_method":
        FD004_RELIABILITY_PHASE_QUANTILE_METHOD_630N,

    "phase_assignment_right":
        bool(FD004_RELIABILITY_PHASE_DIGITIZE_RIGHT_630N),

    "phase_edges_sha256":
        FD004_PHASE_EDGES_SHA256_630O,

    "fit_phase_ids_sha256":
        FD004_FIT_PHASE_IDS_SHA256_630O,

    "phase_counts":
        [
            int(x)
            for x in FD004_FIT_PHASE_COUNTS_630O
        ],

    "phase_reference_sha256":
        list(FD004_PHASE_MC_STD_REFERENCE_SHA256_630O),

    "dev_targets_or_errors_used":
        False,

    "dev_risk_created":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "model_state_unchanged":
        True,

    "model_mode_restored":
        True,
}

FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_630O = _sha_json_630o(
    FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_630O
)


# =============================================================================
# 11) Final safety + lock
# =============================================================================

model_sha_final_630o = _sha_model_state_630o(
    MODEL_FD004_630I
)

if model_sha_final_630o != model_sha_before_630o:
    raise RuntimeError(
        "Model state changed before STEP 6.30O lock."
    )

if MODEL_FD004_630I.training != pre_execution_training_mode_630o:
    raise RuntimeError(
        "Model mode changed before STEP 6.30O lock."
    )

FD004_FIT_CALIBRATION_EXECUTION_COMPLETED_630O = True
FD004_FIT_CALIBRATION_INFERENCE_PERFORMED_630O = True
FD004_FIT_MC_CALIBRATION_PERFORMED_630O = True
FD004_FIT_PHASE_REFERENCE_LOCKED_630O = True

FD004_DEV_OUTCOMES_USED_630O = False
FD004_DEV_RISK_CREATED_630O = False

FD004_THRESHOLD_SELECTED_630O = False
FD004_POLICY_SELECTED_630O = False

FD004_TEST_CONTENTS_READ_630O = False
FD004_RUL_CONTENTS_READ_630O = False

all_checks_passed_630o = True


# =============================================================================
# 12) Report
# =============================================================================

fit_std_summary_630o = (
    float(np.min(FD004_FIT_MC_STD_630O)),
    float(np.median(FD004_FIT_MC_STD_630O)),
    float(np.mean(FD004_FIT_MC_STD_630O, dtype=np.float64)),
    float(np.max(FD004_FIT_MC_STD_630O)),
)

mean_abs_mc_det_630o = float(
    np.mean(
        np.abs(
            FD004_FIT_MC_MEAN_630O
            -
            FD004_FIT_DETERMINISTIC_PREDICTION_630O.astype(np.float64)
        ),
        dtype=np.float64,
    )
)

print("=" * 168)

print(
    "STEP 6.30O — FD004 FIT-ONLY DETERMINISTIC + CANONICAL MC "
    "CALIBRATION EXECUTION + PHASE-REFERENCE LOCK"
)

print("=" * 168)

print("FIT deterministic inference   :", True)
print("FIT MC calibration executed   :", True)
print("DEV targets/errors used       :", False)
print("DEV reliability risk created  :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- FIT CALIBRATION EXECUTION ---")

print("FIT population                :", n_fit_630o)

print(
    "MC matrix shape / dtype       :",
    (
        FD004_FIT_MC_MATRIX_630O.shape,
        FD004_FIT_MC_MATRIX_630O.dtype,
    ),
)

print(
    "MC passes / batch / seed/ddof :",
    (
        passes_630o,
        batch_size_630o,
        seed_630o,
        ddof_630o,
    ),
)

print(
    "Deterministic runtime seconds :",
    FD004_FIT_DETERMINISTIC_RUNTIME_SECONDS_630O,
)

print(
    "MC runtime seconds            :",
    FD004_FIT_MC_RUNTIME_SECONDS_630O,
)

print(
    "FIT MC STD min/median/mean/max:",
    fit_std_summary_630o,
)

print(
    "Mean |MC mean-deterministic|  :",
    mean_abs_mc_det_630o,
)

print("\n--- FD004 FIT-ONLY PHASE CALIBRATION ---")

print(
    "Phase edges                   :",
    FD004_PHASE_EDGES_630O.tolist(),
)

print(
    "Phase counts                  :",
    FD004_FIT_PHASE_COUNTS_630O.tolist(),
)

print(
    "Reference sizes               :",
    [
        int(ref.shape[0])
        for ref in FD004_PHASE_MC_STD_REFERENCES_630O
    ],
)

print(
    "All phase references sorted   :",
    all(
        bool(np.all(np.diff(ref) >= 0.0))
        for ref in FD004_PHASE_MC_STD_REFERENCES_630O
    ),
)

print("\n--- MODEL SAFETY ---")

print(
    "Model state unchanged         :",
    model_sha_final_630o == model_sha_before_630o,
)

print(
    "Model mode restored           :",
    MODEL_FD004_630I.training == pre_execution_training_mode_630o,
)

print(
    "Final model mode              :",
    "eval" if not MODEL_FD004_630I.training else "train",
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "FIT deterministic pred SHA-256:",
    FD004_FIT_DETERMINISTIC_PREDICTION_SHA256_630O,
)

print(
    "FIT MC matrix SHA-256         :",
    FD004_FIT_MC_MATRIX_SHA256_630O,
)

print(
    "FIT MC STD SHA-256            :",
    FD004_FIT_MC_STD_SHA256_630O,
)

print(
    "Phase edges SHA-256           :",
    FD004_PHASE_EDGES_SHA256_630O,
)

print(
    "Phase-reference SHA-256       :",
    FD004_PHASE_MC_STD_REFERENCE_SHA256_630O,
)

print(
    "Calibration evidence SHA-256  :",
    FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_630O,
)

print(
    "\nPASS — the FD004 FIT-only deterministic/MC calibration execution "
    "is complete and the dataset-specific phase edges plus within-phase "
    "uncertainty references are frozen."
)

print(
    "Next: apply these frozen FIT-only phase references to the already-locked "
    "clean DEV deterministic prediction + MC STD to construct phase-adjusted "
    "uncertainty and G1 risk, still without using DEV labels/errors or selecting "
    "any threshold."
)

In [ ]:
# STEP 6.30P
# FD004 CLEAN-DEV PHASE ADJUSTMENT + G1 RIGHT-ECDF RISK LOCK
#
# NO NEW MODEL INFERENCE.
# NO MC EXECUTION.
# NO DEV TARGET/ERROR USE.
# NO G2 CONSTRUCTION.
# NO THRESHOLD/POLICY SELECTION.
# NO TEST / OFFICIAL RUL ACCESS.

import hashlib
import json
import numpy as np


def _sha_arr_630p(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30P hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_630p(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630o", False)):
    raise RuntimeError("STEP 6.30O must pass before STEP 6.30P.")

if not bool(globals().get("FD004_FIT_PHASE_REFERENCE_LOCKED_630O", False)):
    raise RuntimeError("FD004 FIT phase references are not locked.")

if bool(globals().get("FD004_CLEAN_DEV_G1_LOCKED_630P", False)):
    raise RuntimeError("STEP 6.30P has already been completed.")

required_630p = [
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_PREDICTION_SHA256_630K",
    "FD004_CLEAN_DEV_MC_STD_630M",
    "FD004_CLEAN_DEV_MC_STD_SHA256_630M",
    "FD004_PHASE_EDGES_630O",
    "FD004_PHASE_EDGES_SHA256_630O",
    "FD004_PHASE_MC_STD_REFERENCES_630O",
    "FD004_PHASE_MC_STD_REFERENCE_SHA256_630O",
    "FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N",
    "FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_630O",
]

missing_630p = [
    name
    for name in required_630p
    if name not in globals()
]

if missing_630p:
    raise RuntimeError(
        "Missing required STEP 6.30P object(s): "
        + ", ".join(missing_630p)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630O", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30P.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630O", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30P.")


# =============================================================================
# 2) Re-certify frozen DEV + FIT-calibration authorities
# =============================================================================

dev_prediction_630p = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float32,
)

dev_mc_std_630p = np.asarray(
    FD004_CLEAN_DEV_MC_STD_630M,
    dtype=np.float64,
)

phase_edges_630p = np.asarray(
    FD004_PHASE_EDGES_630O,
    dtype=np.float64,
)

phase_refs_630p = tuple(
    np.asarray(ref, dtype=np.float64)
    for ref in FD004_PHASE_MC_STD_REFERENCES_630O
)

if _sha_arr_630p(dev_prediction_630p) != FD004_CLEAN_DEV_PREDICTION_SHA256_630K:
    raise RuntimeError(
        "Frozen clean-DEV deterministic prediction changed."
    )

if _sha_arr_630p(dev_mc_std_630p) != FD004_CLEAN_DEV_MC_STD_SHA256_630M:
    raise RuntimeError(
        "Frozen clean-DEV MC STD changed."
    )

if _sha_arr_630p(phase_edges_630p) != FD004_PHASE_EDGES_SHA256_630O:
    raise RuntimeError(
        "Frozen FD004 phase edges changed."
    )

if len(phase_refs_630p) != 5:
    raise RuntimeError(
        "Expected exactly five frozen phase references."
    )

for idx_630p, ref_630p in enumerate(phase_refs_630p):
    if (
        _sha_arr_630p(ref_630p)
        != FD004_PHASE_MC_STD_REFERENCE_SHA256_630O[idx_630p]
    ):
        raise RuntimeError(
            f"Frozen phase-reference fingerprint changed for phase {idx_630p}."
        )

if dev_prediction_630p.shape != (10824,):
    raise RuntimeError(
        "Unexpected clean-DEV deterministic prediction shape."
    )

if dev_mc_std_630p.shape != (10824,):
    raise RuntimeError(
        "Unexpected clean-DEV MC-STD shape."
    )

if phase_edges_630p.shape != (4,):
    raise RuntimeError(
        "Unexpected phase-edge shape."
    )

if not np.all(np.isfinite(dev_prediction_630p)):
    raise RuntimeError(
        "Clean-DEV deterministic prediction contains non-finite values."
    )

if not np.all(np.isfinite(dev_mc_std_630p)):
    raise RuntimeError(
        "Clean-DEV MC STD contains non-finite values."
    )


# =============================================================================
# 3) Frozen phase-adjustment helper
# =============================================================================

def apply_fd004_phase_adjustment_630p(
    deterministic_prediction,
    mc_std,
):
    prediction_local_630p = np.asarray(
        deterministic_prediction,
        dtype=np.float64,
    ).reshape(-1)

    std_local_630p = np.asarray(
        mc_std,
        dtype=np.float64,
    ).reshape(-1)

    if prediction_local_630p.shape != std_local_630p.shape:
        raise RuntimeError(
            "Prediction and MC-STD vectors differ in shape."
        )

    phase_ids_local_630p = np.digitize(
        prediction_local_630p,
        phase_edges_630p,
        right=True,
    ).astype(
        np.int64,
        copy=False,
    )

    adjusted_local_630p = np.empty(
        std_local_630p.shape[0],
        dtype=np.float64,
    )

    for phase_id_local_630p in range(5):

        mask_local_630p = (
            phase_ids_local_630p
            == phase_id_local_630p
        )

        ref_local_630p = phase_refs_630p[
            phase_id_local_630p
        ]

        n_ref_local_630p = int(
            ref_local_630p.shape[0]
        )

        if n_ref_local_630p <= 0:
            raise RuntimeError(
                f"Empty phase reference encountered for phase "
                f"{phase_id_local_630p}."
            )

        values_local_630p = std_local_630p[
            mask_local_630p
        ]

        left_local_630p = np.searchsorted(
            ref_local_630p,
            values_local_630p,
            side="left",
        )

        right_local_630p = np.searchsorted(
            ref_local_630p,
            values_local_630p,
            side="right",
        )

        exact_tie_local_630p = (
            right_local_630p
            > left_local_630p
        )

        scores_local_630p = (
            left_local_630p.astype(np.float64)
            / float(n_ref_local_630p)
        )

        if np.any(exact_tie_local_630p):

            scores_local_630p[
                exact_tie_local_630p
            ] = (
                (
                    left_local_630p[
                        exact_tie_local_630p
                    ].astype(np.float64)
                    +
                    right_local_630p[
                        exact_tie_local_630p
                    ].astype(np.float64)
                )
                /
                (2.0 * float(n_ref_local_630p))
            )

        adjusted_local_630p[
            mask_local_630p
        ] = scores_local_630p

    if not np.all(np.isfinite(adjusted_local_630p)):
        raise RuntimeError(
            "Phase-adjusted uncertainty contains non-finite values."
        )

    if np.any(
        (adjusted_local_630p < 0.0)
        |
        (adjusted_local_630p > 1.0)
    ):
        raise RuntimeError(
            "Phase-adjusted uncertainty escaped [0, 1]."
        )

    return (
        phase_ids_local_630p,
        adjusted_local_630p,
    )


# =============================================================================
# 4) Apply frozen FIT-only phase references to clean DEV
# =============================================================================

(
    dev_phase_ids_630p,
    dev_adjusted_uncertainty_630p,
) = apply_fd004_phase_adjustment_630p(
    dev_prediction_630p,
    dev_mc_std_630p,
)

dev_phase_counts_630p = np.bincount(
    dev_phase_ids_630p,
    minlength=5,
).astype(
    np.int64,
    copy=False,
)

if int(np.sum(dev_phase_counts_630p)) != 10824:
    raise RuntimeError(
        "Clean-DEV phase counts do not sum to the DEV population."
    )

if np.any(dev_phase_counts_630p <= 0):
    raise RuntimeError(
        "At least one frozen prediction phase is absent in clean DEV."
    )


# =============================================================================
# 5) Construct clean-DEV G1 reference + right-ECDF risk
# =============================================================================

FD004_G1_REFERENCE_630P = np.sort(
    dev_adjusted_uncertainty_630p.astype(
        np.float64,
        copy=True,
    ),
    kind="mergesort",
)

FD004_CLEAN_DEV_G1_RISK_630P = (
    np.searchsorted(
        FD004_G1_REFERENCE_630P,
        dev_adjusted_uncertainty_630p,
        side="right",
    ).astype(np.float64)
    /
    float(FD004_G1_REFERENCE_630P.shape[0])
)

if FD004_G1_REFERENCE_630P.shape != (10824,):
    raise RuntimeError(
        "Unexpected G1 reference shape."
    )

if FD004_CLEAN_DEV_G1_RISK_630P.shape != (10824,):
    raise RuntimeError(
        "Unexpected clean-DEV G1-risk shape."
    )

if not np.all(
    np.diff(FD004_G1_REFERENCE_630P) >= 0.0
):
    raise RuntimeError(
        "G1 reference is not sorted ascending."
    )

if not np.all(
    np.isfinite(FD004_CLEAN_DEV_G1_RISK_630P)
):
    raise RuntimeError(
        "Clean-DEV G1 risk contains non-finite values."
    )

if np.any(
    (FD004_CLEAN_DEV_G1_RISK_630P <= 0.0)
    |
    (FD004_CLEAN_DEV_G1_RISK_630P > 1.0)
):
    raise RuntimeError(
        "Right-ECDF G1 risk escaped the expected interval (0, 1]."
    )


# =============================================================================
# 6) Freeze clean-DEV phase/G1 artifacts
# =============================================================================

FD004_CLEAN_DEV_PHASE_IDS_630P = np.array(
    dev_phase_ids_630p,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_PHASE_COUNTS_630P = np.array(
    dev_phase_counts_630p,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P = np.array(
    dev_adjusted_uncertainty_630p,
    dtype=np.float64,
    copy=True,
    order="C",
)

for arr_630p in (
    FD004_CLEAN_DEV_PHASE_IDS_630P,
    FD004_CLEAN_DEV_PHASE_COUNTS_630P,
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P,
    FD004_G1_REFERENCE_630P,
    FD004_CLEAN_DEV_G1_RISK_630P,
):
    arr_630p.setflags(write=False)


# =============================================================================
# 7) Fingerprints
# =============================================================================

FD004_CLEAN_DEV_PHASE_IDS_SHA256_630P = _sha_arr_630p(
    FD004_CLEAN_DEV_PHASE_IDS_630P
)

FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_630P = _sha_arr_630p(
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P
)

FD004_G1_REFERENCE_SHA256_630P = _sha_arr_630p(
    FD004_G1_REFERENCE_630P
)

FD004_CLEAN_DEV_G1_RISK_SHA256_630P = _sha_arr_630p(
    FD004_CLEAN_DEV_G1_RISK_630P
)


# =============================================================================
# 8) G1 evidence contract
# =============================================================================

FD004_G1_EVIDENCE_CONTRACT_630P = {

    "step":
        "6.30P",

    "dataset":
        "FD004",

    "fit_calibration_evidence_sha256":
        FD004_FIT_CALIBRATION_EVIDENCE_CONTRACT_SHA256_630O,

    "reliability_calibration_protocol_sha256":
        FD004_RELIABILITY_CALIBRATION_PROTOCOL_SHA256_630N,

    "clean_dev_prediction_sha256":
        FD004_CLEAN_DEV_PREDICTION_SHA256_630K,

    "clean_dev_mc_std_sha256":
        FD004_CLEAN_DEV_MC_STD_SHA256_630M,

    "phase_edges_sha256":
        FD004_PHASE_EDGES_SHA256_630O,

    "phase_reference_sha256":
        list(FD004_PHASE_MC_STD_REFERENCE_SHA256_630O),

    "phase_assignment":
        "np.digitize(..., right=True)",

    "phase_adjustment":
        "FIT within-phase empirical midrank percentile",

    "clean_dev_population":
        10824,

    "clean_dev_phase_counts":
        [
            int(x)
            for x in FD004_CLEAN_DEV_PHASE_COUNTS_630P
        ],

    "adjusted_uncertainty_sha256":
        FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_630P,

    "g1_reference_rule":
        "stable ascending clean-DEV adjusted uncertainty",

    "g1_mapping":
        "right empirical CDF",

    "g1_reference_sha256":
        FD004_G1_REFERENCE_SHA256_630P,

    "clean_dev_g1_risk_sha256":
        FD004_CLEAN_DEV_G1_RISK_SHA256_630P,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "dev_targets_or_errors_used":
        False,

    "g2_created":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_G1_EVIDENCE_CONTRACT_SHA256_630P = _sha_json_630p(
    FD004_G1_EVIDENCE_CONTRACT_630P
)


# =============================================================================
# 9) State lock
# =============================================================================

FD004_CLEAN_DEV_G1_LOCKED_630P = True

FD004_NEW_MODEL_INFERENCE_PERFORMED_630P = False
FD004_NEW_MC_EXECUTION_PERFORMED_630P = False

FD004_DEV_OUTCOMES_USED_630P = False
FD004_G2_CREATED_630P = False

FD004_THRESHOLD_SELECTED_630P = False
FD004_POLICY_SELECTED_630P = False

FD004_TEST_CONTENTS_READ_630P = False
FD004_RUL_CONTENTS_READ_630P = False

all_checks_passed_630p = True


# =============================================================================
# 10) Report
# =============================================================================

adjusted_summary_630p = (
    float(
        np.min(
            FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P
        )
    ),
    float(
        np.median(
            FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P
        )
    ),
    float(
        np.mean(
            FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P,
            dtype=np.float64,
        )
    ),
    float(
        np.max(
            FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P
        )
    ),
)

g1_summary_630p = (
    float(np.min(FD004_CLEAN_DEV_G1_RISK_630P)),
    float(np.median(FD004_CLEAN_DEV_G1_RISK_630P)),
    float(
        np.mean(
            FD004_CLEAN_DEV_G1_RISK_630P,
            dtype=np.float64,
        )
    ),
    float(np.max(FD004_CLEAN_DEV_G1_RISK_630P)),
)

print("=" * 168)

print(
    "STEP 6.30P — FD004 CLEAN-DEV PHASE ADJUSTMENT "
    "+ G1 RIGHT-ECDF RISK LOCK"
)

print("=" * 168)

print("New model inference performed :", False)
print("New MC execution performed    :", False)
print("DEV targets/errors used       :", False)
print("G2 created                    :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- CLEAN DEV PHASE ADJUSTMENT ---")

print(
    "DEV population                :",
    10824,
)

print(
    "DEV phase counts              :",
    FD004_CLEAN_DEV_PHASE_COUNTS_630P.tolist(),
)

print(
    "Adjusted uncertainty min/median/mean/max:",
    adjusted_summary_630p,
)

print("\n--- CLEAN DEV G1 ---")

print(
    "G1 reference size             :",
    FD004_G1_REFERENCE_630P.shape[0],
)

print(
    "G1 unique reference values    :",
    int(
        np.unique(
            FD004_G1_REFERENCE_630P
        ).shape[0]
    ),
)

print(
    "Clean DEV G1 risk min/median/mean/max:",
    g1_summary_630p,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "DEV phase IDs SHA-256         :",
    FD004_CLEAN_DEV_PHASE_IDS_SHA256_630P,
)

print(
    "Adjusted uncertainty SHA-256  :",
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_SHA256_630P,
)

print(
    "G1 reference SHA-256          :",
    FD004_G1_REFERENCE_SHA256_630P,
)

print(
    "Clean DEV G1 risk SHA-256     :",
    FD004_CLEAN_DEV_G1_RISK_SHA256_630P,
)

print(
    "G1 evidence contract SHA-256  :",
    FD004_G1_EVIDENCE_CONTRACT_SHA256_630P,
)

print(
    "\nPASS — the frozen FIT-only phase calibration has been applied to "
    "the already-locked clean DEV prediction/MC-STD evidence, and the "
    "label-free clean-DEV G1 risk is now frozen."
)

print(
    "Next: derive the FD004 clean-DEV equal-engine G2 weights from the "
    "frozen DEV window-unit membership, then construct the weighted-midrank "
    "G2 reference/final selective risk. No DEV labels/errors or threshold "
    "will be used."
)

In [ ]:
# STEP 6.30Q-0
# FD004 CLEAN-DEV G2 INPUT-AUTHORITY LOCATOR
#
# METADATA-ONLY.
# NO INFERENCE / MC.
# NO DEV TARGETS OR ERRORS.
# NO G2 WEIGHTS OR G2 RISK.
# NO THRESHOLD / POLICY.
# NO TEST / OFFICIAL RUL.

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_630p", False)):
    raise RuntimeError("STEP 6.30P must pass before STEP 6.30Q-0.")

if not bool(globals().get("FD004_CLEAN_DEV_G1_LOCKED_630P", False)):
    raise RuntimeError("Frozen clean-DEV G1 risk is not available.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630P", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30Q-0.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630P", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30Q-0.")

candidate_names_630q0 = [
    "FD004_DEV_WINDOW_UNIT_IDS_630G",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "FD004_DEV_WINDOW_ENGINE_IDS_630G",
    "FD004_DEV_ENGINE_IDS_630C",
    "DEV_ENGINE_IDS_FD004_630C",
    "FD004_DEV_RAW_630C",
    "FD004_CLEAN_DEV_G1_RISK_630P",
]

rows_630q0 = []

for name_630q0 in candidate_names_630q0:
    present_630q0 = name_630q0 in globals()

    if not present_630q0:
        rows_630q0.append(
            {
                "name": name_630q0,
                "present": False,
                "type": None,
                "shape_or_len": None,
                "dtype": None,
                "value_preview": None,
            }
        )
        continue

    obj_630q0 = globals()[name_630q0]

    if isinstance(obj_630q0, np.ndarray):
        shape_or_len_630q0 = tuple(obj_630q0.shape)
        dtype_630q0 = str(obj_630q0.dtype)

        if obj_630q0.ndim == 1 and obj_630q0.size <= 300:
            preview_630q0 = obj_630q0.tolist()
        elif obj_630q0.ndim == 1:
            preview_630q0 = {
                "first_10": obj_630q0[:10].tolist(),
                "last_10": obj_630q0[-10:].tolist(),
                "unique_count": int(np.unique(obj_630q0).size),
            }
        else:
            preview_630q0 = None

    elif isinstance(obj_630q0, pd.DataFrame):
        shape_or_len_630q0 = tuple(obj_630q0.shape)
        dtype_630q0 = None
        preview_630q0 = {
            "columns": list(obj_630q0.columns),
        }

    elif isinstance(obj_630q0, (list, tuple)):
        shape_or_len_630q0 = len(obj_630q0)
        dtype_630q0 = None
        preview_630q0 = (
            list(obj_630q0)
            if len(obj_630q0) <= 300
            else {
                "first_10": list(obj_630q0[:10]),
                "last_10": list(obj_630q0[-10:]),
            }
        )

    else:
        try:
            shape_or_len_630q0 = len(obj_630q0)
        except Exception:
            shape_or_len_630q0 = None

        dtype_630q0 = None
        preview_630q0 = repr(obj_630q0)

    rows_630q0.append(
        {
            "name": name_630q0,
            "present": True,
            "type": type(obj_630q0).__name__,
            "shape_or_len": shape_or_len_630q0,
            "dtype": dtype_630q0,
            "value_preview": preview_630q0,
        }
    )

FD004_G2_INPUT_AUTHORITY_LOCATOR_630Q0 = pd.DataFrame(rows_630q0)

print("=" * 168)
print("STEP 6.30Q-0 — FD004 CLEAN-DEV G2 INPUT-AUTHORITY LOCATOR")
print("=" * 168)

print("New model inference performed :", False)
print("New MC execution performed    :", False)
print("DEV targets/errors used       :", False)
print("G2 weights created            :", False)
print("G2 risk created               :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- WHITELISTED G2 INPUT AUTHORITY CANDIDATES ---")
print(FD004_G2_INPUT_AUTHORITY_LOCATOR_630Q0.to_string(index=False))

print(
    "\nPASS — only explicitly whitelisted FD004 clean-DEV membership-authority "
    "candidates were inspected. No G2 state was created or modified."
)

In [ ]:
# STEP 6.30Q
# FD004 CLEAN-DEV EQUAL-ENGINE G2 WEIGHT CONSTRUCTION + INPUT-CONTRACT LOCK
#
# NO MODEL INFERENCE.
# NO MC EXECUTION.
# NO DEV TARGET/ERROR USE.
# NO G2 REFERENCE OR FINAL G2 RISK YET.
# NO THRESHOLD/POLICY SELECTION.
# NO TEST / OFFICIAL RUL ACCESS.

import hashlib
import json
import numpy as np


def _sha_arr_630q(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30Q hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_630q(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630p", False)):
    raise RuntimeError("STEP 6.30P must pass before STEP 6.30Q.")

if not bool(globals().get("FD004_CLEAN_DEV_G1_LOCKED_630P", False)):
    raise RuntimeError("Frozen clean-DEV G1 risk is not available.")

if bool(globals().get("FD004_G2_INPUT_CONTRACT_LOCKED_630Q", False)):
    raise RuntimeError("STEP 6.30Q has already been completed.")

required_630q = [
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "FD004_DEV_ENGINE_IDS_630C",
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "FD004_CLEAN_DEV_G1_RISK_SHA256_630P",
    "FD004_G1_EVIDENCE_CONTRACT_SHA256_630P",
]

missing_630q = [
    name
    for name in required_630q
    if name not in globals()
]

if missing_630q:
    raise RuntimeError(
        "Missing required STEP 6.30Q object(s): "
        + ", ".join(missing_630q)
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630P", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30Q.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630P", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30Q.")


# =============================================================================
# 2) Re-certify authoritative clean-DEV membership + G1 alignment
# =============================================================================

window_unit_ids_630q = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

dev_engine_ids_630q = np.asarray(
    FD004_DEV_ENGINE_IDS_630C,
    dtype=np.int64,
).reshape(-1)

clean_g1_630q = np.asarray(
    FD004_CLEAN_DEV_G1_RISK_630P,
    dtype=np.float64,
).reshape(-1)

if window_unit_ids_630q.shape != (10824,):
    raise RuntimeError(
        "Unexpected FD004 DEV window-unit membership shape."
    )

if clean_g1_630q.shape != (10824,):
    raise RuntimeError(
        "Unexpected FD004 clean-DEV G1-risk shape."
    )

if _sha_arr_630q(clean_g1_630q) != FD004_CLEAN_DEV_G1_RISK_SHA256_630P:
    raise RuntimeError(
        "Frozen clean-DEV G1 risk changed before G2 weighting."
    )

unique_window_engines_630q = np.unique(
    window_unit_ids_630q
)

if unique_window_engines_630q.shape != (50,):
    raise RuntimeError(
        "Expected exactly 50 unique DEV engines in the window membership."
    )

if not np.array_equal(
    unique_window_engines_630q,
    np.sort(dev_engine_ids_630q),
):
    raise RuntimeError(
        "DEV window-unit membership does not match the frozen DEV-engine set."
    )

if np.unique(dev_engine_ids_630q).shape[0] != 50:
    raise RuntimeError(
        "Frozen FD004 DEV-engine vector does not contain 50 unique engines."
    )


# =============================================================================
# 3) Derive exact per-engine window counts
# =============================================================================

unique_engines_630q, counts_630q = np.unique(
    window_unit_ids_630q,
    return_counts=True,
)

if unique_engines_630q.shape != (50,):
    raise RuntimeError(
        "Unexpected number of DEV engines while deriving G2 weights."
    )

if int(np.sum(counts_630q)) != 10824:
    raise RuntimeError(
        "DEV per-engine window counts do not sum to 10,824."
    )

if np.any(counts_630q <= 0):
    raise RuntimeError(
        "At least one DEV engine has zero windows."
    )

count_map_630q = {
    int(engine_630q): int(count_630q)
    for engine_630q, count_630q in zip(
        unique_engines_630q,
        counts_630q,
    )
}


# =============================================================================
# 4) Construct equal-engine raw G2 weights
# =============================================================================

n_dev_engines_630q = int(
    unique_engines_630q.shape[0]
)

weights_630q = np.empty(
    window_unit_ids_630q.shape[0],
    dtype=np.float64,
)

for engine_630q in unique_engines_630q:
    engine_int_630q = int(engine_630q)
    n_windows_engine_630q = int(
        count_map_630q[engine_int_630q]
    )

    mask_engine_630q = (
        window_unit_ids_630q
        == engine_int_630q
    )

    weights_630q[
        mask_engine_630q
    ] = (
        1.0
        /
        (
            float(n_dev_engines_630q)
            * float(n_windows_engine_630q)
        )
    )

if not np.all(np.isfinite(weights_630q)):
    raise RuntimeError(
        "Derived equal-engine G2 weights contain non-finite values."
    )

if np.any(weights_630q <= 0.0):
    raise RuntimeError(
        "Derived equal-engine G2 weights must all be positive."
    )

weight_sum_630q = float(
    np.sum(weights_630q, dtype=np.float64)
)

if not np.isclose(
    weight_sum_630q,
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError(
        "Equal-engine G2 weights do not sum to one."
    )


# =============================================================================
# 5) Verify exact equal total mass per DEV engine
# =============================================================================

per_engine_mass_630q = {}

for engine_630q in unique_engines_630q:
    engine_int_630q = int(engine_630q)

    mass_630q = float(
        np.sum(
            weights_630q[
                window_unit_ids_630q
                == engine_int_630q
            ],
            dtype=np.float64,
        )
    )

    per_engine_mass_630q[
        engine_int_630q
    ] = mass_630q

expected_engine_mass_630q = (
    1.0 / float(n_dev_engines_630q)
)

engine_mass_values_630q = np.asarray(
    list(per_engine_mass_630q.values()),
    dtype=np.float64,
)

if not np.allclose(
    engine_mass_values_630q,
    expected_engine_mass_630q,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError(
        "Equal-engine weighting failed: engine masses are not all 1/50."
    )


# =============================================================================
# 6) Freeze G2 input-authority artifacts
# =============================================================================

FD004_DEV_WINDOW_UNIT_IDS_630Q = np.array(
    window_unit_ids_630q,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q = np.array(
    weights_630q,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_DEV_ENGINE_COUNT_MAP_630Q = dict(
    count_map_630q
)

FD004_DEV_ENGINE_TOTAL_MASS_MAP_630Q = dict(
    per_engine_mass_630q
)

FD004_DEV_WINDOW_UNIT_IDS_630Q.setflags(
    write=False
)

FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q.setflags(
    write=False
)


# =============================================================================
# 7) Fingerprints
# =============================================================================

FD004_DEV_WINDOW_UNIT_IDS_SHA256_630Q = _sha_arr_630q(
    FD004_DEV_WINDOW_UNIT_IDS_630Q
)

FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q = _sha_arr_630q(
    FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q
)


# =============================================================================
# 8) G2 input contract
# =============================================================================

FD004_G2_INPUT_CONTRACT_630Q = {

    "step":
        "6.30Q",

    "dataset":
        "FD004",

    "g1_evidence_contract_sha256":
        FD004_G1_EVIDENCE_CONTRACT_SHA256_630P,

    "clean_dev_g1_risk_sha256":
        FD004_CLEAN_DEV_G1_RISK_SHA256_630P,

    "window_membership_source":
        "DEV_WINDOW_UNIT_IDS_FD004_630G",

    "window_population":
        int(FD004_DEV_WINDOW_UNIT_IDS_630Q.shape[0]),

    "dev_engine_count":
        n_dev_engines_630q,

    "dev_engine_ids":
        [
            int(x)
            for x in unique_engines_630q
        ],

    "per_engine_window_counts":
        {
            str(k): int(v)
            for k, v in FD004_DEV_ENGINE_COUNT_MAP_630Q.items()
        },

    "weight_formula":
        (
            "1 / (number_of_DEV_engines * "
            "number_of_windows_for_that_engine)"
        ),

    "expected_total_mass_per_engine":
        expected_engine_mass_630q,

    "raw_weight_sum":
        weight_sum_630q,

    "window_unit_ids_sha256":
        FD004_DEV_WINDOW_UNIT_IDS_SHA256_630Q,

    "raw_equal_engine_weights_sha256":
        FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q,

    "dev_targets_or_errors_used":
        False,

    "g2_reference_created":
        False,

    "final_selective_risk_created":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_G2_INPUT_CONTRACT_SHA256_630Q = _sha_json_630q(
    FD004_G2_INPUT_CONTRACT_630Q
)


# =============================================================================
# 9) State lock
# =============================================================================

FD004_G2_INPUT_CONTRACT_LOCKED_630Q = True

FD004_G2_WEIGHTS_CREATED_630Q = True
FD004_G2_REFERENCE_CREATED_630Q = False
FD004_FINAL_SELECTIVE_RISK_CREATED_630Q = False

FD004_NEW_MODEL_INFERENCE_PERFORMED_630Q = False
FD004_NEW_MC_EXECUTION_PERFORMED_630Q = False
FD004_DEV_OUTCOMES_USED_630Q = False

FD004_THRESHOLD_SELECTED_630Q = False
FD004_POLICY_SELECTED_630Q = False

FD004_TEST_CONTENTS_READ_630Q = False
FD004_RUL_CONTENTS_READ_630Q = False

all_checks_passed_630q = True


# =============================================================================
# 10) Report
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30Q — FD004 CLEAN-DEV EQUAL-ENGINE "
    "G2 WEIGHT CONSTRUCTION + INPUT-CONTRACT LOCK"
)

print("=" * 168)

print("New model inference performed :", False)
print("New MC execution performed    :", False)
print("DEV targets/errors used       :", False)
print("G2 weights created            :", True)
print("G2 reference created          :", False)
print("Final selective risk created  :", False)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- AUTHORITATIVE CLEAN-DEV MEMBERSHIP ---")

print(
    "Window-unit source            :",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
)

print(
    "DEV windows                   :",
    FD004_DEV_WINDOW_UNIT_IDS_630Q.shape[0],
)

print(
    "DEV engines                   :",
    n_dev_engines_630q,
)

print(
    "Engine IDs                    :",
    unique_engines_630q.tolist(),
)

print(
    "Matches frozen DEV split      :",
    np.array_equal(
        unique_engines_630q,
        np.sort(dev_engine_ids_630q),
    ),
)

print("\n--- EQUAL-ENGINE G2 WEIGHTS ---")

print(
    "Per-engine window counts      :",
    FD004_DEV_ENGINE_COUNT_MAP_630Q,
)

print(
    "Weight sum                    :",
    weight_sum_630q,
)

print(
    "Unique raw weights            :",
    int(
        np.unique(
            FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q
        ).shape[0]
    ),
)

print(
    "Expected mass / engine        :",
    expected_engine_mass_630q,
)

print(
    "Per-engine mass min/max       :",
    (
        float(np.min(engine_mass_values_630q)),
        float(np.max(engine_mass_values_630q)),
    ),
)

print(
    "All engine masses exact 1/50  :",
    bool(
        np.allclose(
            engine_mass_values_630q,
            expected_engine_mass_630q,
            rtol=0.0,
            atol=1e-14,
        )
    ),
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "DEV window IDs SHA-256        :",
    FD004_DEV_WINDOW_UNIT_IDS_SHA256_630Q,
)

print(
    "Raw G2 weights SHA-256        :",
    FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q,
)

print(
    "G2 input contract SHA-256     :",
    FD004_G2_INPUT_CONTRACT_SHA256_630Q,
)

print(
    "\nPASS — the FD004 clean-DEV equal-engine G2 weighting vector is "
    "derived from the frozen window membership and locked without using "
    "DEV outcomes."
)

print(
    "Next: pair these frozen weights with the frozen clean-DEV G1 risk, "
    "stable-sort them, aggregate ties, and construct the weighted-midrank "
    "G2 reference and final clean-DEV selective risk. No threshold or "
    "DEV outcome information will be used."
)

In [ ]:
# STEP 6.30R
# FD004 CLEAN-DEV WEIGHTED-MIDRANK G2 REFERENCE + FINAL SELECTIVE-RISK LOCK
#
# NO MODEL INFERENCE.
# NO MC EXECUTION.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD/POLICY SELECTION.
# NO TEST / OFFICIAL RUL ACCESS.

import hashlib
import json
import numpy as np


def _sha_arr_630r(value):
    arr = np.ascontiguousarray(np.asarray(value))
    if arr.dtype.hasobject:
        raise RuntimeError("Object-dtype array refused during STEP 6.30R hashing.")
    return hashlib.sha256(arr.tobytes(order="C")).hexdigest()


def _sha_json_630r(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630q", False)):
    raise RuntimeError("STEP 6.30Q must pass before STEP 6.30R.")

if not bool(globals().get("FD004_G2_INPUT_CONTRACT_LOCKED_630Q", False)):
    raise RuntimeError("FD004 G2 input contract is not locked.")

if bool(globals().get("FD004_G2_REFERENCE_LOCKED_630R", False)):
    raise RuntimeError("STEP 6.30R has already been completed.")

required_630r = [
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "FD004_CLEAN_DEV_G1_RISK_SHA256_630P",
    "FD004_G1_EVIDENCE_CONTRACT_SHA256_630P",
    "FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q",
    "FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q",
    "FD004_G2_INPUT_CONTRACT_SHA256_630Q",
]

missing_630r = [
    name
    for name in required_630r
    if name not in globals()
]

if missing_630r:
    raise RuntimeError(
        "Missing required STEP 6.30R object(s): "
        + ", ".join(missing_630r)
    )

if bool(globals().get("FD004_DEV_OUTCOMES_USED_630Q", True)):
    raise RuntimeError("DEV outcomes were unexpectedly used before STEP 6.30R.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630Q", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30R.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630Q", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30R.")


# =============================================================================
# 2) Re-certify frozen G1 + paired equal-engine weights
# =============================================================================

clean_g1_630r = np.asarray(
    FD004_CLEAN_DEV_G1_RISK_630P,
    dtype=np.float64,
).reshape(-1)

raw_weights_630r = np.asarray(
    FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q,
    dtype=np.float64,
).reshape(-1)

if clean_g1_630r.shape != (10824,):
    raise RuntimeError("Unexpected clean-DEV G1-risk shape.")

if raw_weights_630r.shape != (10824,):
    raise RuntimeError("Unexpected raw G2-weight shape.")

if _sha_arr_630r(clean_g1_630r) != FD004_CLEAN_DEV_G1_RISK_SHA256_630P:
    raise RuntimeError("Frozen clean-DEV G1 risk changed before G2 construction.")

if (
    _sha_arr_630r(raw_weights_630r)
    != FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q
):
    raise RuntimeError("Frozen equal-engine G2 weights changed.")

if not np.all(np.isfinite(clean_g1_630r)):
    raise RuntimeError("Clean-DEV G1 risk contains non-finite values.")

if not np.all(np.isfinite(raw_weights_630r)):
    raise RuntimeError("Raw G2 weights contain non-finite values.")

if np.any(raw_weights_630r <= 0.0):
    raise RuntimeError("Raw G2 weights must all be positive.")

if not np.isclose(
    np.sum(raw_weights_630r, dtype=np.float64),
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError("Raw equal-engine G2 weights no longer sum to one.")


# =============================================================================
# 3) Stable-sort G1 values while preserving paired weights
# =============================================================================

sort_idx_630r = np.argsort(
    clean_g1_630r,
    kind="mergesort",
)

reference_values_630r = clean_g1_630r[
    sort_idx_630r
].astype(
    np.float64,
    copy=True,
)

reference_weights_630r = raw_weights_630r[
    sort_idx_630r
].astype(
    np.float64,
    copy=True,
)

if not np.all(np.diff(reference_values_630r) >= 0.0):
    raise RuntimeError("G2 reference values are not sorted ascending.")

if not np.isclose(
    np.sum(reference_weights_630r, dtype=np.float64),
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError("Sorted G2 reference weights do not sum to one.")


# =============================================================================
# 4) Aggregate exact G1 ties into weighted unique-value masses
# =============================================================================

(
    unique_values_630r,
    unique_inverse_630r,
) = np.unique(
    reference_values_630r,
    return_inverse=True,
)

unique_weights_630r = np.bincount(
    unique_inverse_630r,
    weights=reference_weights_630r,
    minlength=unique_values_630r.shape[0],
).astype(
    np.float64,
    copy=False,
)

if unique_values_630r.shape[0] <= 0:
    raise RuntimeError("No unique G2 reference values were created.")

if unique_weights_630r.shape != unique_values_630r.shape:
    raise RuntimeError("Unique G2 value/weight shapes do not match.")

if np.any(unique_weights_630r <= 0.0):
    raise RuntimeError("Unique G2 masses must all be positive.")

if not np.isclose(
    np.sum(unique_weights_630r, dtype=np.float64),
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError("Aggregated unique G2 masses do not sum to one.")

cumulative_weights_630r = np.cumsum(
    unique_weights_630r,
    dtype=np.float64,
)

mid_mass_630r = (
    cumulative_weights_630r
    - 0.5 * unique_weights_630r
)

if np.any(np.diff(cumulative_weights_630r) <= 0.0):
    raise RuntimeError("G2 cumulative masses are not strictly increasing.")

if not np.isclose(
    cumulative_weights_630r[-1],
    1.0,
    rtol=0.0,
    atol=1e-14,
):
    raise RuntimeError("Final G2 cumulative mass is not one.")

if np.any(
    (mid_mass_630r <= 0.0)
    |
    (mid_mass_630r >= 1.0)
):
    raise RuntimeError("Weighted G2 mid-masses escaped the open interval (0,1).")


# =============================================================================
# 5) Frozen weighted-midrank G2 mapping
# =============================================================================

def apply_fd004_g2_weighted_midrank_630r(values):
    x_630r = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)

    if not np.all(np.isfinite(x_630r)):
        raise RuntimeError("G2 input contains non-finite values.")

    result_630r = np.empty(
        x_630r.shape[0],
        dtype=np.float64,
    )

    left_630r = np.searchsorted(
        unique_values_630r,
        x_630r,
        side="left",
    )

    right_630r = np.searchsorted(
        unique_values_630r,
        x_630r,
        side="right",
    )

    n_unique_630r = int(
        unique_values_630r.shape[0]
    )

    for i_630r in range(x_630r.shape[0]):
        li_630r = int(left_630r[i_630r])
        ri_630r = int(right_630r[i_630r])

        # Exact reference tie -> weighted mid-mass.
        if ri_630r > li_630r:
            result_630r[i_630r] = mid_mass_630r[li_630r]

        # Below minimum reference.
        elif li_630r == 0:
            result_630r[i_630r] = 0.0

        # Above maximum reference.
        elif li_630r >= n_unique_630r:
            result_630r[i_630r] = 1.0

        # Between reference values -> cumulative mass strictly below x.
        else:
            result_630r[i_630r] = cumulative_weights_630r[
                li_630r - 1
            ]

    if not np.all(np.isfinite(result_630r)):
        raise RuntimeError("Weighted-midrank G2 output contains non-finite values.")

    if np.any(
        (result_630r < 0.0)
        |
        (result_630r > 1.0)
    ):
        raise RuntimeError("Weighted-midrank G2 output escaped [0,1].")

    return result_630r


# =============================================================================
# 6) Apply G2 mapping to frozen clean-DEV G1 risk
# =============================================================================

clean_final_risk_630r = apply_fd004_g2_weighted_midrank_630r(
    clean_g1_630r
)

if clean_final_risk_630r.shape != (10824,):
    raise RuntimeError("Unexpected final clean-DEV selective-risk shape.")

expected_sorted_risk_630r = mid_mass_630r[
    unique_inverse_630r
]

observed_sorted_risk_630r = clean_final_risk_630r[
    sort_idx_630r
]

if not np.array_equal(
    observed_sorted_risk_630r,
    expected_sorted_risk_630r,
):
    raise RuntimeError(
        "Weighted-midrank G2 reference semantics failed exact tie check."
    )


# =============================================================================
# 7) Freeze G2 reference + final clean-DEV selective risk
# =============================================================================

FD004_G2_SORT_INDEX_630R = np.array(
    sort_idx_630r,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_G2_REFERENCE_VALUES_630R = np.array(
    reference_values_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_G2_REFERENCE_WEIGHTS_630R = np.array(
    reference_weights_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_G2_UNIQUE_VALUES_630R = np.array(
    unique_values_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_G2_UNIQUE_INVERSE_630R = np.array(
    unique_inverse_630r,
    dtype=np.int64,
    copy=True,
    order="C",
)

FD004_G2_UNIQUE_WEIGHTS_630R = np.array(
    unique_weights_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_G2_CUMULATIVE_WEIGHTS_630R = np.array(
    cumulative_weights_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_G2_MID_MASS_630R = np.array(
    mid_mass_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R = np.array(
    clean_final_risk_630r,
    dtype=np.float64,
    copy=True,
    order="C",
)

for arr_630r in (
    FD004_G2_SORT_INDEX_630R,
    FD004_G2_REFERENCE_VALUES_630R,
    FD004_G2_REFERENCE_WEIGHTS_630R,
    FD004_G2_UNIQUE_VALUES_630R,
    FD004_G2_UNIQUE_INVERSE_630R,
    FD004_G2_UNIQUE_WEIGHTS_630R,
    FD004_G2_CUMULATIVE_WEIGHTS_630R,
    FD004_G2_MID_MASS_630R,
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
):
    arr_630r.setflags(write=False)


# =============================================================================
# 8) Fingerprints
# =============================================================================

FD004_G2_REFERENCE_VALUES_SHA256_630R = _sha_arr_630r(
    FD004_G2_REFERENCE_VALUES_630R
)

FD004_G2_REFERENCE_WEIGHTS_SHA256_630R = _sha_arr_630r(
    FD004_G2_REFERENCE_WEIGHTS_630R
)

FD004_G2_UNIQUE_VALUES_SHA256_630R = _sha_arr_630r(
    FD004_G2_UNIQUE_VALUES_630R
)

FD004_G2_UNIQUE_WEIGHTS_SHA256_630R = _sha_arr_630r(
    FD004_G2_UNIQUE_WEIGHTS_630R
)

FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_630R = _sha_arr_630r(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R
)


# =============================================================================
# 9) Final clean selective-risk evidence contract
# =============================================================================

FD004_CLEAN_SELECTIVE_RISK_CONTRACT_630R = {

    "step":
        "6.30R",

    "dataset":
        "FD004",

    "g1_evidence_contract_sha256":
        FD004_G1_EVIDENCE_CONTRACT_SHA256_630P,

    "g2_input_contract_sha256":
        FD004_G2_INPUT_CONTRACT_SHA256_630Q,

    "clean_dev_g1_risk_sha256":
        FD004_CLEAN_DEV_G1_RISK_SHA256_630P,

    "raw_equal_engine_weights_sha256":
        FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_SHA256_630Q,

    "reference_population":
        10824,

    "reference_sort":
        "stable np.argsort(kind='mergesort')",

    "paired_weights_preserved":
        True,

    "tie_aggregation":
        "np.unique(sorted_G1, return_inverse=True) + weighted bincount",

    "mapping":
        (
            "exact tie -> weighted mid-mass; below minimum -> 0; "
            "above maximum -> 1; otherwise cumulative mass strictly below x"
        ),

    "g2_reference_values_sha256":
        FD004_G2_REFERENCE_VALUES_SHA256_630R,

    "g2_reference_weights_sha256":
        FD004_G2_REFERENCE_WEIGHTS_SHA256_630R,

    "g2_unique_values_sha256":
        FD004_G2_UNIQUE_VALUES_SHA256_630R,

    "g2_unique_weights_sha256":
        FD004_G2_UNIQUE_WEIGHTS_SHA256_630R,

    "final_clean_dev_selective_risk_sha256":
        FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_630R,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "dev_targets_or_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_630R = _sha_json_630r(
    FD004_CLEAN_SELECTIVE_RISK_CONTRACT_630R
)


# =============================================================================
# 10) State lock
# =============================================================================

FD004_G2_REFERENCE_LOCKED_630R = True
FD004_G2_REFERENCE_CREATED_630R = True
FD004_FINAL_SELECTIVE_RISK_CREATED_630R = True

FD004_NEW_MODEL_INFERENCE_PERFORMED_630R = False
FD004_NEW_MC_EXECUTION_PERFORMED_630R = False
FD004_DEV_OUTCOMES_USED_630R = False

FD004_THRESHOLD_SELECTED_630R = False
FD004_POLICY_SELECTED_630R = False

FD004_TEST_CONTENTS_READ_630R = False
FD004_RUL_CONTENTS_READ_630R = False

all_checks_passed_630r = True


# =============================================================================
# 11) Report
# =============================================================================

final_risk_summary_630r = (
    float(np.min(FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R)),
    float(np.median(FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R)),
    float(
        np.mean(
            FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
            dtype=np.float64,
        )
    ),
    float(np.max(FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R)),
)

print("=" * 168)

print(
    "STEP 6.30R — FD004 CLEAN-DEV WEIGHTED-MIDRANK "
    "G2 REFERENCE + FINAL SELECTIVE-RISK LOCK"
)

print("=" * 168)

print("New model inference performed :", False)
print("New MC execution performed    :", False)
print("DEV targets/errors used       :", False)
print("G2 reference created          :", True)
print("Final selective risk created  :", True)
print("Threshold selected            :", False)
print("Policy selected               :", False)
print("TEST contents read            :", False)
print("Official RUL contents read    :", False)

print("\n--- WEIGHTED G2 REFERENCE ---")

print(
    "Reference windows             :",
    FD004_G2_REFERENCE_VALUES_630R.shape[0],
)

print(
    "Unique G1 values              :",
    FD004_G2_UNIQUE_VALUES_630R.shape[0],
)

print(
    "Tie-group count               :",
    FD004_G2_UNIQUE_VALUES_630R.shape[0],
)

print(
    "Reference weight sum          :",
    float(
        np.sum(
            FD004_G2_REFERENCE_WEIGHTS_630R,
            dtype=np.float64,
        )
    ),
)

print(
    "Unique grouped-weight sum     :",
    float(
        np.sum(
            FD004_G2_UNIQUE_WEIGHTS_630R,
            dtype=np.float64,
        )
    ),
)

print(
    "Exact reference midmass check :",
    bool(
        np.array_equal(
            observed_sorted_risk_630r,
            expected_sorted_risk_630r,
        )
    ),
)

print("\n--- FINAL CLEAN-DEV SELECTIVE RISK ---")

print(
    "Risk population               :",
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R.shape[0],
)

print(
    "Risk min/median/mean/max      :",
    final_risk_summary_630r,
)

print(
    "Risk within [0,1]             :",
    bool(
        np.all(
            (FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R >= 0.0)
            &
            (FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R <= 1.0)
        )
    ),
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "G2 reference values SHA-256   :",
    FD004_G2_REFERENCE_VALUES_SHA256_630R,
)

print(
    "G2 reference weights SHA-256  :",
    FD004_G2_REFERENCE_WEIGHTS_SHA256_630R,
)

print(
    "G2 unique values SHA-256      :",
    FD004_G2_UNIQUE_VALUES_SHA256_630R,
)

print(
    "G2 unique weights SHA-256     :",
    FD004_G2_UNIQUE_WEIGHTS_SHA256_630R,
)

print(
    "Final selective risk SHA-256  :",
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_SHA256_630R,
)

print(
    "Selective-risk contract SHA-256:",
    FD004_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_630R,
)

print(
    "\nPASS — the FD004 clean-DEV weighted-midrank G2 reference and final "
    "label-free selective-risk vector are now constructed, fingerprinted, "
    "and locked."
)

print(
    "No DEV outcomes/errors or threshold were used. Next: prospectively "
    "freeze the FD004 controlled-corruption experiment design before any "
    "corruption execution or DEV outcome-based threshold/reporting analysis."
)

In [ ]:
# STEP 6.30S
# FD004 CONTROLLED-CORRUPTION DESIGN + 441-SCENARIO MANIFEST PROSPECTIVE FREEZE
#
# DESIGN / METADATA ONLY.
# NO CORRUPTION EXECUTION.
# NO MODEL INFERENCE.
# NO MC EXECUTION.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD/POLICY SELECTION.
# NO TEST / OFFICIAL RUL ACCESS.

import hashlib
import json
import numpy as np
import pandas as pd


def _sha_json_630s(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def _canonical_df_records_630s(df):
    records_630s = []

    for row_630s in df.to_dict(orient="records"):
        clean_row_630s = {}

        for key_630s, value_630s in row_630s.items():

            if pd.isna(value_630s):
                clean_row_630s[key_630s] = None

            elif isinstance(value_630s, np.integer):
                clean_row_630s[key_630s] = int(value_630s)

            elif isinstance(value_630s, np.floating):
                clean_row_630s[key_630s] = float(value_630s)

            else:
                clean_row_630s[key_630s] = value_630s

        records_630s.append(clean_row_630s)

    return records_630s


def _sha_df_630s(df):
    return _sha_json_630s(
        _canonical_df_records_630s(df)
    )


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630r", False)):
    raise RuntimeError("STEP 6.30R must pass before STEP 6.30S.")

if not bool(globals().get("FD004_G2_REFERENCE_LOCKED_630R", False)):
    raise RuntimeError("FD004 clean-DEV G2 reference is not locked.")

if not bool(globals().get("FD004_FINAL_SELECTIVE_RISK_CREATED_630R", False)):
    raise RuntimeError("FD004 final clean-DEV selective risk is not locked.")

if bool(globals().get("FD004_CORRUPTION_DESIGN_LOCKED_630S", False)):
    raise RuntimeError("STEP 6.30S has already been completed.")

required_630s = [
    "FD004_SENSOR_COLS_630E",
    "FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2",
    "FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2",
    "FD004_PREPROCESSING_PROTOCOL_SHA256_630E",
    "FD004_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_630R",
]

missing_630s = [
    name
    for name in required_630s
    if name not in globals()
]

if missing_630s:
    raise RuntimeError(
        "Missing required STEP 6.30S object(s): "
        + ", ".join(missing_630s)
    )

if bool(globals().get("FD004_DEV_OUTCOMES_USED_630R", True)):
    raise RuntimeError(
        "DEV outcomes were unexpectedly used before STEP 6.30S."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630R", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30S."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630R", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30S."
    )


# =============================================================================
# 2) Freeze authoritative physical-sensor corruption scope
# =============================================================================

sensor_scope_630s = tuple(
    str(x)
    for x in FD004_SENSOR_COLS_630E
)

expected_sensor_scope_630s = tuple(
    f"sensor_{i}"
    for i in range(1, 22)
)

if sensor_scope_630s != expected_sensor_scope_630s:
    raise RuntimeError(
        "FD004 corruption sensor scope is not exactly sensor_1..sensor_21 "
        "in original order."
    )

if len(sensor_scope_630s) != 21:
    raise RuntimeError(
        "Expected exactly 21 FD004 physical sensors."
    )

authoritative_features_630s = tuple(
    str(x)
    for x in FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2
)

expected_features_630s = (
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
) + expected_sensor_scope_630s

if authoritative_features_630s != expected_features_630s:
    raise RuntimeError(
        "Authoritative FD004 raw feature order changed before corruption freeze."
    )


# =============================================================================
# 3) Freeze transferred corruption semantics
# =============================================================================
#
# These are methodology semantics only.
# No prior-dataset numerical scales, means, thresholds, risks, or outcomes
# are inherited.
#

FD004_CORRUPTION_FAMILY_ORDER_630S = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

FD004_BIAS_LEVELS_630S = (
    ("low",    0.25, -1.0),
    ("low",    0.25,  1.0),
    ("medium", 0.50, -1.0),
    ("medium", 0.50,  1.0),
    ("high",   1.00, -1.0),
    ("high",   1.00,  1.0),
)

FD004_NOISE_LEVELS_630S = (
    ("low",    0.10),
    ("medium", 0.25),
    ("high",   0.50),
)

FD004_DRIFT_LEVELS_630S = (
    ("low",    0.25, -1.0),
    ("low",    0.25,  1.0),
    ("medium", 0.50, -1.0),
    ("medium", 0.50,  1.0),
    ("high",   1.00, -1.0),
    ("high",   1.00,  1.0),
)

FD004_DROPOUT_LEVELS_630S = (
    ("low",    0.10),
    ("medium", 0.25),
    ("high",   0.50),
)

FD004_STUCK_LEVELS_630S = (
    ("low",    0.20),
    ("medium", 0.40),
    ("high",   0.60),
)

FD004_CORRUPTION_RANDOM_SEED_630S = 42

FD004_CORRUPTION_SCENARIO_ID_RULE_630S = (
    "FD004_S{scenario_number:03d}"
)

FD004_CORRUPTION_ORDER_RULE_630S = (
    "family_block_then_sensor_then_within_family_level_order"
)

FD004_CORRUPTION_NUMERIC_INHERITANCE_630S = False


# =============================================================================
# 4) Construct prospective 441-scenario manifest
# =============================================================================

rows_630s = []
scenario_number_630s = 1


def _append_630s(
    family,
    sensor,
    severity,
    alpha=None,
    direction=None,
    requested_fraction=None,
    seed=None,
):
    global scenario_number_630s

    rows_630s.append(
        {
            "scenario_id":
                FD004_CORRUPTION_SCENARIO_ID_RULE_630S.format(
                    scenario_number=scenario_number_630s
                ),

            "scenario_number":
                int(scenario_number_630s),

            "family":
                str(family),

            "sensor":
                str(sensor),

            "severity":
                str(severity),

            "alpha":
                (
                    np.nan
                    if alpha is None
                    else float(alpha)
                ),

            "direction":
                (
                    np.nan
                    if direction is None
                    else float(direction)
                ),

            "requested_fraction":
                (
                    np.nan
                    if requested_fraction is None
                    else float(requested_fraction)
                ),

            "seed":
                (
                    np.nan
                    if seed is None
                    else int(seed)
                ),
        }
    )

    scenario_number_630s += 1


# Bias block: 6 scenarios / sensor.
for sensor_630s in sensor_scope_630s:
    for severity_630s, alpha_630s, direction_630s in FD004_BIAS_LEVELS_630S:
        _append_630s(
            family="bias",
            sensor=sensor_630s,
            severity=severity_630s,
            alpha=alpha_630s,
            direction=direction_630s,
        )


# Noise block: 3 scenarios / sensor.
for sensor_630s in sensor_scope_630s:
    for severity_630s, alpha_630s in FD004_NOISE_LEVELS_630S:
        _append_630s(
            family="noise",
            sensor=sensor_630s,
            severity=severity_630s,
            alpha=alpha_630s,
            seed=FD004_CORRUPTION_RANDOM_SEED_630S,
        )


# Drift block: 6 scenarios / sensor.
for sensor_630s in sensor_scope_630s:
    for severity_630s, alpha_630s, direction_630s in FD004_DRIFT_LEVELS_630S:
        _append_630s(
            family="drift",
            sensor=sensor_630s,
            severity=severity_630s,
            alpha=alpha_630s,
            direction=direction_630s,
        )


# Dropout block: 3 scenarios / sensor.
for sensor_630s in sensor_scope_630s:
    for severity_630s, fraction_630s in FD004_DROPOUT_LEVELS_630S:
        _append_630s(
            family="dropout",
            sensor=sensor_630s,
            severity=severity_630s,
            requested_fraction=fraction_630s,
            seed=FD004_CORRUPTION_RANDOM_SEED_630S,
        )


# Stuck block: 3 scenarios / sensor.
for sensor_630s in sensor_scope_630s:
    for severity_630s, fraction_630s in FD004_STUCK_LEVELS_630S:
        _append_630s(
            family="stuck",
            sensor=sensor_630s,
            severity=severity_630s,
            requested_fraction=fraction_630s,
        )


FD004_CORRUPTION_MANIFEST_630S = pd.DataFrame(
    rows_630s,
    columns=[
        "scenario_id",
        "scenario_number",
        "family",
        "sensor",
        "severity",
        "alpha",
        "direction",
        "requested_fraction",
        "seed",
    ],
)


# =============================================================================
# 5) Exact manifest validation
# =============================================================================

if FD004_CORRUPTION_MANIFEST_630S.shape != (441, 9):
    raise RuntimeError(
        "FD004 corruption manifest must contain exactly 441 rows and 9 columns."
    )

expected_numbers_630s = np.arange(
    1,
    442,
    dtype=np.int64,
)

actual_numbers_630s = FD004_CORRUPTION_MANIFEST_630S[
    "scenario_number"
].to_numpy(dtype=np.int64)

if not np.array_equal(
    actual_numbers_630s,
    expected_numbers_630s,
):
    raise RuntimeError(
        "FD004 corruption scenario numbers are not exactly contiguous 1..441."
    )

expected_ids_630s = [
    f"FD004_S{i:03d}"
    for i in range(1, 442)
]

if (
    FD004_CORRUPTION_MANIFEST_630S["scenario_id"].tolist()
    != expected_ids_630s
):
    raise RuntimeError(
        "FD004 corruption scenario-ID sequence is incorrect."
    )

family_counts_630s = (
    FD004_CORRUPTION_MANIFEST_630S[
        "family"
    ]
    .value_counts(sort=False)
    .to_dict()
)

expected_family_counts_630s = {
    "bias": 126,
    "noise": 63,
    "drift": 126,
    "dropout": 63,
    "stuck": 63,
}

if family_counts_630s != expected_family_counts_630s:
    raise RuntimeError(
        "FD004 corruption family counts are incorrect."
    )

sensor_counts_630s = (
    FD004_CORRUPTION_MANIFEST_630S[
        "sensor"
    ]
    .value_counts(sort=False)
)

if set(sensor_counts_630s.index) != set(sensor_scope_630s):
    raise RuntimeError(
        "FD004 corruption manifest sensor scope is incomplete or contaminated."
    )

if not np.all(
    sensor_counts_630s.to_numpy(dtype=np.int64) == 21
):
    raise RuntimeError(
        "Each FD004 sensor must have exactly 21 corruption scenarios."
    )


# Verify family blocks are contiguous and in the frozen order.

family_sequence_630s = (
    FD004_CORRUPTION_MANIFEST_630S[
        "family"
    ].tolist()
)

expected_family_sequence_630s = (
    ["bias"] * 126
    + ["noise"] * 63
    + ["drift"] * 126
    + ["dropout"] * 63
    + ["stuck"] * 63
)

if family_sequence_630s != expected_family_sequence_630s:
    raise RuntimeError(
        "FD004 corruption family-block ordering is incorrect."
    )


# Verify exact within-family sensor-outer ordering.

for family_630s in FD004_CORRUPTION_FAMILY_ORDER_630S:

    family_df_630s = (
        FD004_CORRUPTION_MANIFEST_630S[
            FD004_CORRUPTION_MANIFEST_630S["family"]
            == family_630s
        ]
        .reset_index(drop=True)
    )

    per_sensor_expected_630s = (
        6
        if family_630s in ("bias", "drift")
        else 3
    )

    expected_sensor_sequence_630s = []

    for sensor_630s in sensor_scope_630s:
        expected_sensor_sequence_630s.extend(
            [sensor_630s] * per_sensor_expected_630s
        )

    if family_df_630s["sensor"].tolist() != expected_sensor_sequence_630s:
        raise RuntimeError(
            f"Sensor-outer ordering failed for family {family_630s}."
        )


# =============================================================================
# 6) Freeze design contract
# =============================================================================

FD004_CORRUPTION_DESIGN_PROTOCOL_630S = {

    "step":
        "6.30S",

    "dataset":
        "FD004",

    "feature_contract_resolution_sha256":
        FD004_FEATURE_CONTRACT_RESOLUTION_SHA256_630ER2,

    "preprocessing_protocol_sha256":
        FD004_PREPROCESSING_PROTOCOL_SHA256_630E,

    "clean_selective_risk_contract_sha256":
        FD004_CLEAN_SELECTIVE_RISK_CONTRACT_SHA256_630R,

    "methodology_transfer_only":
        True,

    "prior_dataset_numeric_references_inherited":
        False,

    "corruption_scope":
        "physical sensors only",

    "sensor_order":
        list(sensor_scope_630s),

    "sensor_count":
        21,

    "family_order":
        list(FD004_CORRUPTION_FAMILY_ORDER_630S),

    "within_family_order":
        "sensor_outer_then_level_order",

    "bias":
        [
            {
                "severity": sev,
                "alpha": float(alpha),
                "direction": float(direction),
            }
            for sev, alpha, direction
            in FD004_BIAS_LEVELS_630S
        ],

    "noise":
        [
            {
                "severity": sev,
                "alpha": float(alpha),
                "seed": 42,
            }
            for sev, alpha
            in FD004_NOISE_LEVELS_630S
        ],

    "drift":
        [
            {
                "severity": sev,
                "alpha": float(alpha),
                "direction": float(direction),
            }
            for sev, alpha, direction
            in FD004_DRIFT_LEVELS_630S
        ],

    "dropout":
        [
            {
                "severity": sev,
                "requested_fraction": float(frac),
                "seed": 42,
            }
            for sev, frac
            in FD004_DROPOUT_LEVELS_630S
        ],

    "stuck":
        [
            {
                "severity": sev,
                "requested_fraction": float(frac),
            }
            for sev, frac
            in FD004_STUCK_LEVELS_630S
        ],

    "scenarios_per_sensor":
        21,

    "total_scenarios":
        441,

    "scenario_id_rule":
        FD004_CORRUPTION_SCENARIO_ID_RULE_630S,

    "corruption_executed":
        False,

    "dev_targets_or_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_CORRUPTION_DESIGN_PROTOCOL_SHA256_630S = _sha_json_630s(
    FD004_CORRUPTION_DESIGN_PROTOCOL_630S
)

FD004_CORRUPTION_MANIFEST_SHA256_630S = _sha_df_630s(
    FD004_CORRUPTION_MANIFEST_630S
)


# =============================================================================
# 7) State lock
# =============================================================================

FD004_CORRUPTION_DESIGN_LOCKED_630S = True
FD004_CORRUPTION_MANIFEST_LOCKED_630S = True

FD004_CORRUPTION_EXECUTED_630S = False
FD004_CORRUPTION_REFERENCES_CREATED_630S = False

FD004_NEW_MODEL_INFERENCE_PERFORMED_630S = False
FD004_NEW_MC_EXECUTION_PERFORMED_630S = False
FD004_DEV_OUTCOMES_USED_630S = False

FD004_THRESHOLD_SELECTED_630S = False
FD004_POLICY_SELECTED_630S = False

FD004_TEST_CONTENTS_READ_630S = False
FD004_RUL_CONTENTS_READ_630S = False

all_checks_passed_630s = True


# =============================================================================
# 8) Report
# =============================================================================

family_summary_630s = (
    FD004_CORRUPTION_MANIFEST_630S
    .groupby(
        "family",
        sort=False,
    )
    .agg(
        row_count=("scenario_number", "size"),
        first_row=("scenario_number", "min"),
        last_row=("scenario_number", "max"),
    )
    .reset_index()
)

print("=" * 168)

print(
    "STEP 6.30S — FD004 CONTROLLED-CORRUPTION DESIGN "
    "+ 441-SCENARIO MANIFEST PROSPECTIVE FREEZE"
)

print("=" * 168)

print("Corruption design locked       :", True)
print("Corruption manifest locked     :", True)
print("Corruption references created  :", False)
print("Corruption executed            :", False)
print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV targets/errors used        :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print("\n--- FROZEN SENSOR SCOPE ---")

print(
    "Physical sensor count          :",
    len(sensor_scope_630s),
)

print(
    "Physical sensors               :",
    list(sensor_scope_630s),
)

print(
    "Operating settings corrupted   :",
    False,
)

print("\n--- FROZEN CORRUPTION FAMILY DESIGN ---")

print(family_summary_630s.to_string(index=False))

print(
    "\nScenarios per sensor            :",
    21,
)

print(
    "Total scenarios                :",
    FD004_CORRUPTION_MANIFEST_630S.shape[0],
)

print(
    "Scenario-number range          :",
    (
        int(
            FD004_CORRUPTION_MANIFEST_630S[
                "scenario_number"
            ].min()
        ),
        int(
            FD004_CORRUPTION_MANIFEST_630S[
                "scenario_number"
            ].max()
        ),
    ),
)

print(
    "Scenario numbers contiguous    :",
    bool(
        np.array_equal(
            actual_numbers_630s,
            expected_numbers_630s,
        )
    ),
)

print("\nFirst 12 manifest rows:")

print(
    FD004_CORRUPTION_MANIFEST_630S
    .head(12)
    .to_string(index=False)
)

print("\nLast 12 manifest rows:")

print(
    FD004_CORRUPTION_MANIFEST_630S
    .tail(12)
    .to_string(index=False)
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Corruption design SHA-256      :",
    FD004_CORRUPTION_DESIGN_PROTOCOL_SHA256_630S,
)

print(
    "Corruption manifest SHA-256    :",
    FD004_CORRUPTION_MANIFEST_SHA256_630S,
)

print(
    "\nPASS — the FD004 controlled-corruption design and dataset-specific "
    "441-scenario manifest are prospectively frozen before deriving any "
    "corruption magnitudes or executing any corruption."
)

print(
    "Next: derive and lock FD004 FIT-only corruption scale/imputation "
    "references using the already-frozen six-regime preprocessing authority. "
    "No corruption will be executed in that step."
)

In [ ]:
# STEP 6.30T-0
# FD004 FIT-ONLY CORRUPTION-REFERENCE AUTHORITY LOCATOR
#
# METADATA / AUTHORITY CHECK ONLY.
# NO REGIME REFITTING.
# NO NORMALIZATION.
# NO CORRUPTION REFERENCES CREATED.
# NO CORRUPTION EXECUTION.
# NO INFERENCE / MC.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD / POLICY.
# NO TEST / OFFICIAL RUL.

import numpy as np
import pandas as pd


# =============================================================================
# 1) Preconditions
# =============================================================================

if not bool(globals().get("all_checks_passed_630s", False)):
    raise RuntimeError("STEP 6.30S must pass before STEP 6.30T-0.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630S", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30T-0.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630S", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30T-0.")


# =============================================================================
# 2) Whitelisted authority candidates only
# =============================================================================

candidate_names_630t0 = [
    "FD004_FIT_REGIME_IDS_630F",
    "FIT_REGIME_IDS_FD004_630F",

    "FD004_DEV_REGIME_IDS_630F",
    "DEV_REGIME_IDS_FD004_630F",

    "FD004_CANONICAL_REGIME_CENTERS_630F",

    "FD004_SENSOR_MEANS_630F",
    "FD004_SENSOR_EFFECTIVE_STDS_630F",

    "FD004_FIT_RAW_630C",
    "FD004_SENSOR_COLS_630E",

    "FD004_CORRUPTION_MANIFEST_630S",
]


rows_630t0 = []


# =============================================================================
# 3) Metadata-only inspection
# =============================================================================

for name_630t0 in candidate_names_630t0:

    present_630t0 = name_630t0 in globals()

    if not present_630t0:

        rows_630t0.append(
            {
                "name": name_630t0,
                "present": False,
                "type": None,
                "shape_or_len": None,
                "dtype": None,
                "metadata": None,
            }
        )

        continue

    obj_630t0 = globals()[name_630t0]

    # -------------------------------------------------------------------------
    # ndarray
    # -------------------------------------------------------------------------

    if isinstance(obj_630t0, np.ndarray):

        shape_or_len_630t0 = tuple(
            obj_630t0.shape
        )

        dtype_630t0 = str(
            obj_630t0.dtype
        )

        metadata_630t0 = None

        # For likely FIT/DEV regime-ID vectors only:
        # report unique IDs + counts, but do not alter anything.
        if (
            obj_630t0.ndim == 1
            and obj_630t0.size in (48975, 12274)
        ):

            (
                unique_630t0,
                counts_630t0,
            ) = np.unique(
                obj_630t0,
                return_counts=True,
            )

            metadata_630t0 = {
                "unique_values":
                    unique_630t0.tolist(),

                "counts":
                    counts_630t0.tolist(),

                "min":
                    int(np.min(obj_630t0)),

                "max":
                    int(np.max(obj_630t0)),
            }

        elif obj_630t0.size <= 200:

            metadata_630t0 = (
                obj_630t0.tolist()
            )

    # -------------------------------------------------------------------------
    # DataFrame
    # -------------------------------------------------------------------------

    elif isinstance(obj_630t0, pd.DataFrame):

        shape_or_len_630t0 = tuple(
            obj_630t0.shape
        )

        dtype_630t0 = None

        metadata_630t0 = {
            "columns":
                list(obj_630t0.columns)
        }

    # -------------------------------------------------------------------------
    # list / tuple
    # -------------------------------------------------------------------------

    elif isinstance(
        obj_630t0,
        (list, tuple),
    ):

        shape_or_len_630t0 = len(
            obj_630t0
        )

        dtype_630t0 = None

        metadata_630t0 = list(
            obj_630t0
        )

    # -------------------------------------------------------------------------
    # dict
    # -------------------------------------------------------------------------

    elif isinstance(obj_630t0, dict):

        shape_or_len_630t0 = len(
            obj_630t0
        )

        dtype_630t0 = None

        metadata_630t0 = {
            "keys":
                list(obj_630t0.keys())
        }

    # -------------------------------------------------------------------------
    # fallback
    # -------------------------------------------------------------------------

    else:

        try:
            shape_or_len_630t0 = len(
                obj_630t0
            )
        except Exception:
            shape_or_len_630t0 = None

        dtype_630t0 = None
        metadata_630t0 = repr(
            obj_630t0
        )

    rows_630t0.append(
        {
            "name":
                name_630t0,

            "present":
                True,

            "type":
                type(obj_630t0).__name__,

            "shape_or_len":
                shape_or_len_630t0,

            "dtype":
                dtype_630t0,

            "metadata":
                metadata_630t0,
        }
    )


FD004_CORRUPTION_REFERENCE_AUTHORITY_LOCATOR_630T0 = pd.DataFrame(
    rows_630t0
)


# =============================================================================
# 4) Report
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30T-0 — FD004 FIT-ONLY CORRUPTION-REFERENCE "
    "AUTHORITY LOCATOR"
)

print("=" * 168)

print("Regime fitting performed       :", False)
print("Normalization performed        :", False)
print("Corruption references created  :", False)
print("Corruption executed            :", False)
print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV targets/errors used        :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- WHITELISTED REFERENCE-AUTHORITY CANDIDATES ---"
)

print(
    FD004_CORRUPTION_REFERENCE_AUTHORITY_LOCATOR_630T0.to_string(
        index=False
    )
)

print(
    "\nPASS — only explicitly whitelisted FD004 FIT preprocessing/reference "
    "authority objects were inspected. No preprocessing, corruption, "
    "inference, scoring, or threshold operation was executed."
)

In [ ]:
# STEP 6.30T-0R1
# FD004 6.30F SENSOR-STATISTIC AUTHORITY NAME LOCATOR
#
# NAMES / TYPES / SHAPES ONLY.
# NO NUMERICAL VALUES READ OR RECOMPUTED.
# NO REGIME REFITTING / NORMALIZATION.
# NO CORRUPTION REFERENCES OR EXECUTION.
# NO MODEL INFERENCE / MC.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD / POLICY.
# NO TEST / OFFICIAL RUL.

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_630s", False)):
    raise RuntimeError("STEP 6.30S must pass before STEP 6.30T-0R1.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630S", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30T-0R1.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630S", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30T-0R1.")


# =============================================================================
# 1) Names-only discovery restricted to STEP 6.30F preprocessing namespace
# =============================================================================

tokens_630t0r1 = (
    "SENSOR",
    "STD",
    "MEAN",
    "REGIME",
    "GLOBAL",
    "NORMAL",
    "SCAL",
)

candidate_names_630t0r1 = sorted(
    name
    for name in globals().keys()
    if (
        "630F" in str(name).upper()
        and any(
            token in str(name).upper()
            for token in tokens_630t0r1
        )
    )
)


# =============================================================================
# 2) Metadata only: type / shape / length
# =============================================================================

rows_630t0r1 = []

for name_630t0r1 in candidate_names_630t0r1:
    obj_630t0r1 = globals()[name_630t0r1]

    if isinstance(obj_630t0r1, np.ndarray):
        shape_or_len_630t0r1 = tuple(obj_630t0r1.shape)
        dtype_630t0r1 = str(obj_630t0r1.dtype)

    elif isinstance(obj_630t0r1, pd.DataFrame):
        shape_or_len_630t0r1 = tuple(obj_630t0r1.shape)
        dtype_630t0r1 = None

    elif isinstance(obj_630t0r1, pd.Series):
        shape_or_len_630t0r1 = tuple(obj_630t0r1.shape)
        dtype_630t0r1 = str(obj_630t0r1.dtype)

    elif isinstance(obj_630t0r1, (list, tuple, dict)):
        shape_or_len_630t0r1 = len(obj_630t0r1)
        dtype_630t0r1 = None

    else:
        try:
            shape_or_len_630t0r1 = len(obj_630t0r1)
        except Exception:
            shape_or_len_630t0r1 = None
        dtype_630t0r1 = None

    rows_630t0r1.append(
        {
            "name": name_630t0r1,
            "type": type(obj_630t0r1).__name__,
            "shape_or_len": shape_or_len_630t0r1,
            "dtype": dtype_630t0r1,
        }
    )

FD004_630F_SENSOR_STAT_AUTHORITY_NAME_LOCATOR_630T0R1 = pd.DataFrame(
    rows_630t0r1
)


# =============================================================================
# 3) Report
# =============================================================================

print("=" * 168)
print(
    "STEP 6.30T-0R1 — FD004 6.30F SENSOR-STATISTIC "
    "AUTHORITY NAME LOCATOR"
)
print("=" * 168)

print("Numerical values inspected     :", False)
print("Statistics recomputed          :", False)
print("Regime fitting performed       :", False)
print("Normalization performed        :", False)
print("Corruption references created  :", False)
print("Corruption executed            :", False)
print("New model inference performed  :", False)
print("New MC execution performed     :", False)
print("DEV targets/errors used        :", False)
print("Threshold selected             :", False)
print("Policy selected                :", False)
print("TEST contents read             :", False)
print("Official RUL contents read     :", False)

print(
    "\n--- STEP 6.30F PREPROCESSING-STATISTIC NAMESPACE CANDIDATES ---"
)

if FD004_630F_SENSOR_STAT_AUTHORITY_NAME_LOCATOR_630T0R1.empty:
    print("No matching STEP 6.30F names found.")
else:
    print(
        FD004_630F_SENSOR_STAT_AUTHORITY_NAME_LOCATOR_630T0R1.to_string(
            index=False
        )
    )

print(
    "\nPASS — only object names plus type/shape metadata inside the "
    "STEP 6.30F preprocessing namespace were inspected. "
    "No numerical preprocessing state was read or recomputed."
)

In [ ]:
# STEP 6.30T-R1
# FD004 HISTORICAL FD002 CORRUPTION-REFERENCE FORMULA SOURCE RECOVERY
#
# SOURCE-ONLY FORENSIC READ.
# NO HISTORICAL CELL EXECUTION.
# NO DATASET CONTENTS READ.
# NO REGIME REFITTING / NORMALIZATION.
# NO CORRUPTION REFERENCES CREATED.
# NO CORRUPTION EXECUTION.
# NO MODEL INFERENCE / MC.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD / POLICY.
# NO TEST / OFFICIAL RUL.

import hashlib
import json
from pathlib import Path

if not bool(globals().get("all_checks_passed_630s", False)):
    raise RuntimeError("STEP 6.30S must pass before STEP 6.30T-R1.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630S", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30T-R1.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630S", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30T-R1.")


# =============================================================================
# 1) Historical notebook source authority
# =============================================================================

FD002_NOTEBOOK_PATH_630TR1 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks\07_fd002_replication.ipynb"
)

if not FD002_NOTEBOOK_PATH_630TR1.is_file():
    raise RuntimeError(
        "Historical FD002 notebook was not found at the expected path."
    )

with FD002_NOTEBOOK_PATH_630TR1.open(
    "r",
    encoding="utf-8",
) as f_630tr1:
    nb_630tr1 = json.load(f_630tr1)

cells_630tr1 = nb_630tr1.get("cells", [])

search_terms_630tr1 = (
    "pooled_within_regime",
    "global_raw_std",
    "dropout",
    "imputation",
    "corruption scale",
    "scale reference",
)

matches_630tr1 = []

for cell_index_630tr1, cell_630tr1 in enumerate(cells_630tr1):
    if cell_630tr1.get("cell_type") != "code":
        continue

    source_630tr1 = "".join(cell_630tr1.get("source", []))
    source_lower_630tr1 = source_630tr1.lower()

    matched_terms_630tr1 = [
        term
        for term in search_terms_630tr1
        if term in source_lower_630tr1
    ]

    if not matched_terms_630tr1:
        continue

    strong_630tr1 = (
        "pooled_within_regime" in source_lower_630tr1
        or "global_raw_std" in source_lower_630tr1
        or (
            "dropout" in source_lower_630tr1
            and "imputation" in source_lower_630tr1
        )
    )

    if not strong_630tr1:
        continue

    matches_630tr1.append(
        {
            "cell_index": int(cell_index_630tr1),
            "matched_terms": tuple(matched_terms_630tr1),
            "source": source_630tr1,
            "source_sha256": hashlib.sha256(
                source_630tr1.encode("utf-8")
            ).hexdigest(),
        }
    )


# =============================================================================
# 2) Require a compact, inspectable candidate set
# =============================================================================

if not matches_630tr1:
    raise RuntimeError(
        "No historical FD002 corruption-reference source candidate was found."
    )

if len(matches_630tr1) > 8:
    raise RuntimeError(
        "Historical source search returned more than 8 strong candidates; "
        "stop rather than selecting ambiguously."
    )

FD004_HISTORICAL_CORRUPTION_REFERENCE_SOURCE_CANDIDATES_630TR1 = tuple(
    {
        "cell_index": item_630tr1["cell_index"],
        "matched_terms": item_630tr1["matched_terms"],
        "source_sha256": item_630tr1["source_sha256"],
    }
    for item_630tr1 in matches_630tr1
)

FD004_HISTORICAL_CORRUPTION_REFERENCE_SOURCE_RECOVERY_PERFORMED_630TR1 = True

all_checks_passed_630tr1 = True


# =============================================================================
# 3) Report exact candidate source without executing it
# =============================================================================

print("=" * 168)
print(
    "STEP 6.30T-R1 — FD004 HISTORICAL FD002 CORRUPTION-REFERENCE "
    "FORMULA SOURCE RECOVERY"
)
print("=" * 168)

print("Historical notebook source read :", True)
print("Historical cells executed       :", False)
print("Dataset contents read           :", False)
print("Statistics recomputed           :", False)
print("Regime fitting performed        :", False)
print("Normalization performed         :", False)
print("Corruption references created   :", False)
print("Corruption executed             :", False)
print("Model inference / MC performed  :", False)
print("DEV targets/errors used         :", False)
print("Threshold / policy selected     :", False)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)

print("\nHistorical notebook:")
print(FD002_NOTEBOOK_PATH_630TR1)

print("\nStrong source candidates:", len(matches_630tr1))

for rank_630tr1, item_630tr1 in enumerate(matches_630tr1, start=1):
    print("\n" + "-" * 168)
    print("Candidate rank :", rank_630tr1)
    print("Cell index     :", item_630tr1["cell_index"])
    print("Matched terms  :", list(item_630tr1["matched_terms"]))
    print("Source SHA-256 :", item_630tr1["source_sha256"])
    print("-" * 168)

    source_lines_630tr1 = item_630tr1["source"].splitlines()

    for line_no_630tr1, line_630tr1 in enumerate(
        source_lines_630tr1,
        start=1,
    ):
        print(f"{line_no_630tr1:04d}: {line_630tr1}")

print(
    "\nPASS — historical FD002 notebook code was read as source text only. "
    "No historical cell was executed and no dataset value was read. "
    "Use the displayed source to resolve the exact corruption-reference formula."
)

In [ ]:
# STEP 6.30T-R2
# FD004 TARGETED HISTORICAL FD002 CORRUPTION-REFERENCE FORMULA RECOVERY
#
# SOURCE-ONLY FORENSIC READ.
# NO HISTORICAL CELL EXECUTION.
# NO DATASET CONTENTS READ.
# NO REGIME REFITTING / NORMALIZATION.
# NO CORRUPTION REFERENCES CREATED.
# NO CORRUPTION EXECUTION.
# NO MODEL INFERENCE / MC.
# NO DEV TARGET/ERROR USE.
# NO THRESHOLD / POLICY.
# NO TEST / OFFICIAL RUL.

import ast
import hashlib
import json
import re
from pathlib import Path

if not bool(globals().get("all_checks_passed_630s", False)):
    raise RuntimeError("STEP 6.30S must pass before STEP 6.30T-R2.")

if bool(globals().get("FD004_TEST_CONTENTS_READ_630S", True)):
    raise RuntimeError("TEST was unexpectedly read before STEP 6.30T-R2.")

if bool(globals().get("FD004_RUL_CONTENTS_READ_630S", True)):
    raise RuntimeError("Official RUL was unexpectedly read before STEP 6.30T-R2.")


# =============================================================================
# 1) Historical notebook source authority
# =============================================================================

FD002_NOTEBOOK_PATH_630TR2 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks\07_fd002_replication.ipynb"
)

if not FD002_NOTEBOOK_PATH_630TR2.is_file():
    raise RuntimeError(
        "Historical FD002 notebook was not found at the expected path."
    )

with FD002_NOTEBOOK_PATH_630TR2.open(
    "r",
    encoding="utf-8",
) as f_630tr2:
    nb_630tr2 = json.load(f_630tr2)

cells_630tr2 = nb_630tr2.get("cells", [])


# =============================================================================
# 2) Locate exact known historical corruption-operator definitions
# =============================================================================

target_functions_630tr2 = (
    "apply_fd002_bias_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
)

operator_hits_630tr2 = []

for cell_index_630tr2, cell_630tr2 in enumerate(cells_630tr2):

    if cell_630tr2.get("cell_type") != "code":
        continue

    source_630tr2 = "".join(
        cell_630tr2.get("source", [])
    )

    defined_here_630tr2 = tuple(
        fn_630tr2
        for fn_630tr2 in target_functions_630tr2
        if re.search(
            rf"(?m)^\s*def\s+{re.escape(fn_630tr2)}\s*\(",
            source_630tr2,
        )
    )

    if not defined_here_630tr2:
        continue

    operator_hits_630tr2.append(
        {
            "cell_index": int(cell_index_630tr2),
            "defined_functions": defined_here_630tr2,
            "source": source_630tr2,
            "source_sha256": hashlib.sha256(
                source_630tr2.encode("utf-8")
            ).hexdigest(),
        }
    )

if not operator_hits_630tr2:
    raise RuntimeError(
        "No exact historical FD002 corruption-operator definition cell was found."
    )

if len(operator_hits_630tr2) > 4:
    raise RuntimeError(
        "More than four exact corruption-operator definition cells were found; "
        "stop before selecting an authority."
    )


# =============================================================================
# 3) Extract symbol names referenced inside those exact operator cells
# =============================================================================

referenced_names_630tr2 = set()

for hit_630tr2 in operator_hits_630tr2:

    try:
        tree_630tr2 = ast.parse(
            hit_630tr2["source"]
        )
    except SyntaxError:
        tree_630tr2 = None

    if tree_630tr2 is not None:
        for node_630tr2 in ast.walk(tree_630tr2):
            if isinstance(node_630tr2, ast.Name):
                referenced_names_630tr2.add(
                    node_630tr2.id
                )

reference_tokens_630tr2 = (
    "scale",
    "std",
    "mean",
    "median",
    "imput",
    "fallback",
    "sensor",
    "reference",
)

reference_symbol_candidates_630tr2 = sorted(
    name_630tr2
    for name_630tr2 in referenced_names_630tr2
    if any(
        token_630tr2 in name_630tr2.lower()
        for token_630tr2 in reference_tokens_630tr2
    )
)


# =============================================================================
# 4) Locate assignments/definitions of only those referenced symbols
# =============================================================================

reference_definition_hits_630tr2 = []

for symbol_630tr2 in reference_symbol_candidates_630tr2:

    pattern_630tr2 = re.compile(
        rf"(?m)^\s*{re.escape(symbol_630tr2)}\s*="
    )

    for cell_index_630tr2, cell_630tr2 in enumerate(cells_630tr2):

        if cell_630tr2.get("cell_type") != "code":
            continue

        source_630tr2 = "".join(
            cell_630tr2.get("source", [])
        )

        if pattern_630tr2.search(source_630tr2):

            reference_definition_hits_630tr2.append(
                {
                    "symbol": symbol_630tr2,
                    "cell_index": int(cell_index_630tr2),
                    "source_sha256": hashlib.sha256(
                        source_630tr2.encode("utf-8")
                    ).hexdigest(),
                    "source": source_630tr2,
                }
            )

# Deduplicate identical symbol/cell pairs.
dedup_630tr2 = {}

for hit_630tr2 in reference_definition_hits_630tr2:
    key_630tr2 = (
        hit_630tr2["symbol"],
        hit_630tr2["cell_index"],
    )
    dedup_630tr2[key_630tr2] = hit_630tr2

reference_definition_hits_630tr2 = list(
    dedup_630tr2.values()
)


# =============================================================================
# 5) Freeze source-recovery audit metadata only
# =============================================================================

FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2 = tuple(
    {
        "cell_index": hit_630tr2["cell_index"],
        "defined_functions": hit_630tr2["defined_functions"],
        "source_sha256": hit_630tr2["source_sha256"],
    }
    for hit_630tr2 in operator_hits_630tr2
)

FD004_HISTORICAL_CORRUPTION_REFERENCE_SYMBOLS_630TR2 = tuple(
    reference_symbol_candidates_630tr2
)

FD004_HISTORICAL_CORRUPTION_REFERENCE_DEFINITION_HITS_630TR2 = tuple(
    {
        "symbol": hit_630tr2["symbol"],
        "cell_index": hit_630tr2["cell_index"],
        "source_sha256": hit_630tr2["source_sha256"],
    }
    for hit_630tr2 in reference_definition_hits_630tr2
)

FD004_HISTORICAL_CORRUPTION_SOURCE_RECOVERY_PERFORMED_630TR2 = True

all_checks_passed_630tr2 = True


# =============================================================================
# 6) Report exact source authority, without executing it
# =============================================================================

print("=" * 168)

print(
    "STEP 6.30T-R2 — FD004 TARGETED HISTORICAL FD002 "
    "CORRUPTION-REFERENCE FORMULA RECOVERY"
)

print("=" * 168)

print("Historical notebook source read :", True)
print("Historical cells executed       :", False)
print("Dataset contents read           :", False)
print("Statistics recomputed           :", False)
print("Regime fitting performed        :", False)
print("Normalization performed         :", False)
print("Corruption references created   :", False)
print("Corruption executed             :", False)
print("Model inference / MC performed  :", False)
print("DEV targets/errors used         :", False)
print("Threshold / policy selected     :", False)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)

print("\n--- EXACT HISTORICAL OPERATOR DEFINITION CELLS ---")

for rank_630tr2, hit_630tr2 in enumerate(
    operator_hits_630tr2,
    start=1,
):
    print("\n" + "-" * 168)
    print("Operator-source rank :", rank_630tr2)
    print("Cell index           :", hit_630tr2["cell_index"])
    print("Defined functions    :", list(hit_630tr2["defined_functions"]))
    print("Source SHA-256       :", hit_630tr2["source_sha256"])
    print("-" * 168)

    for line_no_630tr2, line_630tr2 in enumerate(
        hit_630tr2["source"].splitlines(),
        start=1,
    ):
        print(f"{line_no_630tr2:04d}: {line_630tr2}")

print("\n--- REFERENCE-LIKE SYMBOLS USED BY THOSE OPERATORS ---")
print(reference_symbol_candidates_630tr2)

print("\n--- ASSIGNMENT CELLS FOR THOSE SYMBOLS ---")

if not reference_definition_hits_630tr2:
    print("No direct assignment cells found for the extracted symbols.")

else:
    for rank_630tr2, hit_630tr2 in enumerate(
        reference_definition_hits_630tr2,
        start=1,
    ):
        print("\n" + "-" * 168)
        print("Definition-hit rank :", rank_630tr2)
        print("Symbol              :", hit_630tr2["symbol"])
        print("Cell index          :", hit_630tr2["cell_index"])
        print("Source SHA-256      :", hit_630tr2["source_sha256"])
        print("-" * 168)

        for line_no_630tr2, line_630tr2 in enumerate(
            hit_630tr2["source"].splitlines(),
            start=1,
        ):
            print(f"{line_no_630tr2:04d}: {line_630tr2}")

print(
    "\nPASS — the search was restricted to exact historical FD002 corruption "
    "operator definitions and the reference-like symbols they actually use. "
    "Nothing was executed or recomputed."
)

In [ ]:
# STEP 6.30T
# FD004 FIT-ONLY HISTORICAL-SEMANTICS CORRUPTION REFERENCE DERIVATION
# + SCALE-VIABILITY AUDIT
#
# NO corruption execution.
# NO new inference / MC.
# NO DEV labels/errors.
# NO threshold/policy.
# NO TEST / official RUL.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630s", False)):
    raise RuntimeError(
        "STEP 6.30S must pass before STEP 6.30T."
    )

if not bool(globals().get("all_checks_passed_630tr2", False)):
    raise RuntimeError(
        "STEP 6.30T-R2 must pass before STEP 6.30T."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630S", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before STEP 6.30T."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630S", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before STEP 6.30T."
    )


required_630t = [
    "FD004_FIT_RAW_630C",
    "FD004_FIT_REGIME_IDS_630F",
    "FD004_FIT_REGIME_SENSOR_MEANS_630F",
    "FD004_SENSOR_COLS_630E",
    "FD004_CORRUPTION_MANIFEST_630S",
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
    "FD004_HISTORICAL_CORRUPTION_REFERENCE_DEFINITION_HITS_630TR2",
]

missing_630t = [
    name_630t
    for name_630t in required_630t
    if name_630t not in globals()
]

if missing_630t:
    raise RuntimeError(
        "Missing STEP 6.30T authority objects: "
        + str(missing_630t)
    )


# =============================================================================
# 1) VERIFY EXACT HISTORICAL SOURCE AUTHORITY
# =============================================================================

EXPECTED_OPERATOR_SOURCE_SHA256_630T = (
    "74dc1af52e2359a74191cb3121915b89"
    "bb42f4522e1add88e39b1c82e68ced2e"
)

EXPECTED_REFERENCE_SOURCE_SHA256_630T = (
    "7224841c33154c10abc1d992617a0d4e"
    "e95e66411975e5ae4420a24a79aeaa6d"
)


operator_hits_630t = tuple(
    FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2
)

definition_hits_630t = tuple(
    FD004_HISTORICAL_CORRUPTION_REFERENCE_DEFINITION_HITS_630TR2
)


operator_authority_exact_630t = any(
    int(hit_630t.get("cell_index", -1)) == 25
    and
    str(hit_630t.get("source_sha256", ""))
    == EXPECTED_OPERATOR_SOURCE_SHA256_630T
    for hit_630t in operator_hits_630t
)

reference_authority_exact_630t = any(
    int(hit_630t.get("cell_index", -1)) == 23
    and
    str(hit_630t.get("source_sha256", ""))
    == EXPECTED_REFERENCE_SOURCE_SHA256_630T
    for hit_630t in definition_hits_630t
)


if not operator_authority_exact_630t:
    raise RuntimeError(
        "Historical corruption-operator source authority changed."
    )

if not reference_authority_exact_630t:
    raise RuntimeError(
        "Historical corruption-reference source authority changed."
    )


# =============================================================================
# 2) FROZEN FD004 FIT AUTHORITY
# =============================================================================

fit_raw_630t = (
    FD004_FIT_RAW_630C
    .reset_index(drop=True)
)

fit_regime_ids_630t = np.asarray(
    FD004_FIT_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

sensor_cols_630t = tuple(
    str(x_630t)
    for x_630t in FD004_SENSOR_COLS_630E
)


if len(fit_raw_630t) != fit_regime_ids_630t.size:
    raise RuntimeError(
        "FIT raw rows and frozen FIT regime IDs are misaligned."
    )

if len(sensor_cols_630t) != 21:
    raise RuntimeError(
        "Expected 21 FD004 physical sensors, found "
        f"{len(sensor_cols_630t)}."
    )

if tuple(
    sorted(
        np.unique(
            fit_regime_ids_630t
        ).tolist()
    )
) != (
    0, 1, 2, 3, 4, 5
):
    raise RuntimeError(
        "Frozen FD004 FIT regime IDs are not exactly 0..5."
    )


manifest_sensors_630t = tuple(
    pd.unique(
        FD004_CORRUPTION_MANIFEST_630S[
            "sensor"
        ].astype(str)
    )
)

if set(manifest_sensors_630t) != set(sensor_cols_630t):
    raise RuntimeError(
        "Frozen corruption-manifest sensor scope does not "
        "match FD004 sensor authority."
    )


# =============================================================================
# 3) EXACT HISTORICAL-STYLE REGIME × SENSOR STATISTICS
#
# Historical FD002 corruption reference used:
#   regime sensor STD with ddof=1
#
# This is intentionally separate from the STEP 6.30F normalization
# effective-STD convention.
# =============================================================================

stats_rows_630t = []


for regime_id_630t in range(6):

    mask_630t = (
        fit_regime_ids_630t
        ==
        regime_id_630t
    )

    n_rows_630t = int(
        mask_630t.sum()
    )

    if n_rows_630t <= 1:
        raise RuntimeError(
            f"Regime {regime_id_630t} has insufficient "
            "FIT rows for ddof=1."
        )


    for sensor_630t in sensor_cols_630t:

        values_630t = (
            fit_raw_630t.loc[
                mask_630t,
                sensor_630t,
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        mean_630t = float(
            np.mean(
                values_630t
            )
        )

        std_ddof1_630t = float(
            np.std(
                values_630t,
                ddof=1,
            )
        )


        if not np.isfinite(
            mean_630t
        ):
            raise RuntimeError(
                "Non-finite FIT regime mean: "
                f"regime={regime_id_630t}, "
                f"sensor={sensor_630t}."
            )


        if not np.isfinite(
            std_ddof1_630t
        ):
            raise RuntimeError(
                "Non-finite FIT regime sample STD: "
                f"regime={regime_id_630t}, "
                f"sensor={sensor_630t}."
            )


        stats_rows_630t.append(
            {
                "regime_id":
                    int(
                        regime_id_630t
                    ),

                "sensor":
                    sensor_630t,

                "rows":
                    n_rows_630t,

                "mean":
                    mean_630t,

                "std_ddof1":
                    std_ddof1_630t,
            }
        )


FD004_REGIME_SENSOR_CORRUPTION_STATS_630T = (
    pd.DataFrame(
        stats_rows_630t
    )
)


if (
    FD004_REGIME_SENSOR_CORRUPTION_STATS_630T.shape
    !=
    (126, 5)
):
    raise RuntimeError(
        "Expected exactly 6 × 21 = 126 "
        "FD004 regime-sensor reference rows."
    )


# =============================================================================
# 4) RECONFIRM MEANS AGAINST ALREADY-FROZEN STEP 6.30F MEANS
# =============================================================================

frozen_means_630t = np.asarray(
    FD004_FIT_REGIME_SENSOR_MEANS_630F,
    dtype=np.float64,
)

if frozen_means_630t.shape != (
    6,
    21,
):
    raise RuntimeError(
        "Frozen STEP 6.30F regime-sensor mean matrix "
        "has unexpected shape."
    )


derived_means_630t = np.empty(
    (
        6,
        21,
    ),
    dtype=np.float64,
)


for regime_id_630t in range(6):

    for (
        sensor_index_630t,
        sensor_630t,
    ) in enumerate(
        sensor_cols_630t
    ):

        row_630t = (
            FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
                (
                    FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
                        "regime_id"
                    ]
                    ==
                    regime_id_630t
                )
                &
                (
                    FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
                        "sensor"
                    ]
                    ==
                    sensor_630t
                )
            ]
        )

        if len(
            row_630t
        ) != 1:
            raise RuntimeError(
                "Regime-sensor reference lookup "
                "is not one-to-one."
            )


        derived_means_630t[
            regime_id_630t,
            sensor_index_630t,
        ] = float(
            row_630t[
                "mean"
            ].iloc[0]
        )


mean_max_abs_diff_630t = float(
    np.max(
        np.abs(
            derived_means_630t
            -
            frozen_means_630t
        )
    )
)


means_match_frozen_630f_630t = bool(
    np.allclose(
        derived_means_630t,
        frozen_means_630t,
        atol=1e-12,
        rtol=1e-12,
    )
)


if not means_match_frozen_630f_630t:
    raise RuntimeError(
        "Derived FIT regime means do not reproduce "
        "the frozen STEP 6.30F means."
    )


# =============================================================================
# 5) HISTORICAL POOLED WITHIN-REGIME SAMPLE STD
#
# sqrt(
#     sum_r((n_r - 1) * s_r^2)
#     /
#     sum_r(n_r - 1)
# )
# =============================================================================

scale_rows_630t = []


for sensor_630t in sensor_cols_630t:

    sensor_stats_630t = (
        FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
            FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
                "sensor"
            ]
            ==
            sensor_630t
        ]
        .sort_values(
            "regime_id"
        )
        .reset_index(
            drop=True
        )
    )


    n_vec_630t = (
        sensor_stats_630t[
            "rows"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    s_vec_630t = (
        sensor_stats_630t[
            "std_ddof1"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    numerator_630t = float(
        np.sum(
            (
                n_vec_630t
                -
                1.0
            )
            *
            np.square(
                s_vec_630t
            )
        )
    )


    denominator_630t = float(
        np.sum(
            n_vec_630t
            -
            1.0
        )
    )


    if denominator_630t <= 0.0:
        raise RuntimeError(
            "Invalid pooled-STD denominator for "
            f"{sensor_630t}."
        )


    pooled_std_630t = float(
        np.sqrt(
            numerator_630t
            /
            denominator_630t
        )
    )


    global_std_630t = float(
        np.std(
            fit_raw_630t[
                sensor_630t
            ].to_numpy(
                dtype=np.float64
            ),
            ddof=1,
        )
    )


    if (
        not np.isfinite(
            pooled_std_630t
        )
        or
        not np.isfinite(
            global_std_630t
        )
    ):
        raise RuntimeError(
            "Non-finite corruption reference for "
            f"{sensor_630t}."
        )


    if pooled_std_630t > 0.0:

        ratio_630t = float(
            global_std_630t
            /
            pooled_std_630t
        )

    else:

        ratio_630t = np.inf


    scale_rows_630t.append(
        {
            "sensor":
                sensor_630t,

            "pooled_within_regime_std":
                pooled_std_630t,

            "global_raw_std":
                global_std_630t,

            "global_to_within_ratio":
                ratio_630t,
        }
    )


FD004_CORRUPTION_SCALE_REFERENCE_630T = (
    pd.DataFrame(
        scale_rows_630t
    )
)


# =============================================================================
# 6) DROPOUT IMPUTATION AUTHORITY
# =============================================================================

FD004_DROPOUT_IMPUTATION_REFERENCE_630T = (
    FD004_REGIME_SENSOR_CORRUPTION_STATS_630T[
        [
            "regime_id",
            "sensor",
            "mean",
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 7) SCALE VIABILITY AUDIT
#
# IMPORTANT:
# Do NOT invent a fallback in this cell.
# First observe the FIT-only structural fact.
# =============================================================================

FD004_CORRUPTION_SCALE_EPSILON_630T = (
    1e-12
)


FD004_CORRUPTION_SCALE_REFERENCE_630T[
    "positive_gt_0"
] = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T[
        "pooled_within_regime_std"
    ]
    >
    0.0
)


FD004_CORRUPTION_SCALE_REFERENCE_630T[
    "usable_gt_1e_12"
] = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T[
        "pooled_within_regime_std"
    ]
    >
    FD004_CORRUPTION_SCALE_EPSILON_630T
)


zero_scale_sensors_630t = tuple(
    FD004_CORRUPTION_SCALE_REFERENCE_630T.loc[
        ~FD004_CORRUPTION_SCALE_REFERENCE_630T[
            "positive_gt_0"
        ],
        "sensor",
    ]
    .astype(str)
)


nearzero_scale_sensors_630t = tuple(
    FD004_CORRUPTION_SCALE_REFERENCE_630T.loc[
        ~FD004_CORRUPTION_SCALE_REFERENCE_630T[
            "usable_gt_1e_12"
        ],
        "sensor",
    ]
    .astype(str)
)


FD004_ZERO_POOLED_SCALE_SENSORS_630T = (
    zero_scale_sensors_630t
)

FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T = (
    nearzero_scale_sensors_630t
)


FD004_ALL_MAGNITUDE_SCALES_USABLE_630T = bool(
    len(
        FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T
    )
    ==
    0
)


# =============================================================================
# 8) FINGERPRINT HELPERS
# =============================================================================

def sha256_dataframe_630t(
    df_630t
):

    h_630t = hashlib.sha256()

    h_630t.update(
        json.dumps(
            list(
                df_630t.columns
            ),
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )

    row_hashes_630t = (
        pd.util.hash_pandas_object(
            df_630t,
            index=False,
        )
        .to_numpy(
            dtype=np.uint64
        )
    )

    h_630t.update(
        np.ascontiguousarray(
            row_hashes_630t
        ).tobytes()
    )

    return h_630t.hexdigest()


FD004_REGIME_SENSOR_CORRUPTION_STATS_SHA256_630T = (
    sha256_dataframe_630t(
        FD004_REGIME_SENSOR_CORRUPTION_STATS_630T
    )
)


FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T = (
    sha256_dataframe_630t(
        FD004_CORRUPTION_SCALE_REFERENCE_630T
    )
)


FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T = (
    sha256_dataframe_630t(
        FD004_DROPOUT_IMPUTATION_REFERENCE_630T
    )
)


# =============================================================================
# 9) REFERENCE CONTRACT
# =============================================================================

FD004_CORRUPTION_REFERENCE_CONTRACT_630T = {
    "dataset":
        "FD004",

    "source":
        "FIT_only",

    "historical_operator_source_sha256":
        EXPECTED_OPERATOR_SOURCE_SHA256_630T,

    "historical_reference_source_sha256":
        EXPECTED_REFERENCE_SOURCE_SHA256_630T,

    "regime_count":
        6,

    "sensor_count":
        21,

    "regime_std_ddof":
        1,

    "pooled_std_formula":
        (
            "sqrt(sum_r((n_r-1)*std_r_ddof1^2)"
            "/sum_r(n_r-1))"
        ),

    "bias_scale":
        "pooled_within_regime_std",

    "noise_scale":
        "pooled_within_regime_std",

    "drift_scale":
        "pooled_within_regime_std",

    "dropout_imputation":
        "FIT_regime_specific_sensor_mean",

    "stuck_reference_required":
        False,

    "scale_viability_epsilon":
        1e-12,

    "all_magnitude_scales_usable":
        bool(
            FD004_ALL_MAGNITUDE_SCALES_USABLE_630T
        ),

    "nonusable_scale_sensors":
        list(
            FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T
        ),

    "DEV_outcomes_used":
        False,

    "corruption_executed":
        False,

    "threshold_selected":
        False,

    "TEST_read":
        False,

    "official_RUL_read":
        False,
}


FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T = (
    hashlib.sha256(
        json.dumps(
            FD004_CORRUPTION_REFERENCE_CONTRACT_630T,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# =============================================================================
# 10) STATE FLAGS
# =============================================================================

FD004_CORRUPTION_REFERENCES_CREATED_630T = True
FD004_CORRUPTION_EXECUTED_630T = False
FD004_DEV_OUTCOMES_USED_630T = False
FD004_THRESHOLD_SELECTED_630T = False
FD004_POLICY_SELECTED_630T = False
FD004_TEST_CONTENTS_READ_630T = False
FD004_RUL_CONTENTS_READ_630T = False

all_checks_passed_630t = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30T — FD004 FIT-ONLY HISTORICAL-SEMANTICS "
    "CORRUPTION REFERENCE DERIVATION + SCALE-VIABILITY AUDIT"
)

print("=" * 174)


print(
    "Historical operator authority exact :",
    operator_authority_exact_630t,
)

print(
    "Historical reference authority exact:",
    reference_authority_exact_630t,
)

print(
    "Corruption references created       :",
    True,
)

print(
    "Corruption executed                 :",
    False,
)

print(
    "New model inference performed       :",
    False,
)

print(
    "New MC execution performed          :",
    False,
)

print(
    "DEV targets/errors used             :",
    False,
)

print(
    "Threshold selected                  :",
    False,
)

print(
    "Policy selected                     :",
    False,
)

print(
    "TEST contents read                  :",
    False,
)

print(
    "Official RUL contents read          :",
    False,
)


print(
    "\n--- REFERENCE CONSTRUCTION ---"
)

print(
    "FIT rows                         :",
    len(
        fit_raw_630t
    ),
)

print(
    "Operating regimes                :",
    6,
)

print(
    "Physical sensors                 :",
    len(
        sensor_cols_630t
    ),
)

print(
    "Regime × sensor reference rows   :",
    len(
        FD004_REGIME_SENSOR_CORRUPTION_STATS_630T
    ),
)

print(
    "Regime STD ddof                  :",
    1,
)

print(
    "Pooling formula                  : "
    "sqrt(sum((n_r-1)*s_r^2) / sum(n_r-1))"
)

print(
    "Frozen 6.30F means reproduced    :",
    means_match_frozen_630f_630t,
)

print(
    "Mean max absolute difference     :",
    mean_max_abs_diff_630t,
)

print(
    "Dropout imputation rows          :",
    len(
        FD004_DROPOUT_IMPUTATION_REFERENCE_630T
    ),
)


print(
    "\n--- MAGNITUDE-SCALE VIABILITY ---"
)


print(
    "Pooled STD min / median / mean / max:",
    (
        float(
            FD004_CORRUPTION_SCALE_REFERENCE_630T[
                "pooled_within_regime_std"
            ].min()
        ),
        float(
            FD004_CORRUPTION_SCALE_REFERENCE_630T[
                "pooled_within_regime_std"
            ].median()
        ),
        float(
            FD004_CORRUPTION_SCALE_REFERENCE_630T[
                "pooled_within_regime_std"
            ].mean()
        ),
        float(
            FD004_CORRUPTION_SCALE_REFERENCE_630T[
                "pooled_within_regime_std"
            ].max()
        ),
    ),
)


print(
    "Exactly zero-scale sensors        :",
    list(
        FD004_ZERO_POOLED_SCALE_SENSORS_630T
    ),
)

print(
    "Nonusable <= 1e-12 sensors       :",
    list(
        FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T
    ),
)

print(
    "All magnitude scales usable      :",
    FD004_ALL_MAGNITUDE_SCALES_USABLE_630T,
)


print(
    "\nFD004 corruption scale reference:"
)

print(
    FD004_CORRUPTION_SCALE_REFERENCE_630T[
        [
            "sensor",
            "pooled_within_regime_std",
            "global_raw_std",
            "global_to_within_ratio",
            "usable_gt_1e_12",
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\n--- EVIDENCE FINGERPRINTS ---"
)


print(
    "Regime-sensor stats SHA-256       :",
    FD004_REGIME_SENSOR_CORRUPTION_STATS_SHA256_630T,
)

print(
    "Corruption-scale reference SHA-256:",
    FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T,
)

print(
    "Dropout-imputation ref SHA-256    :",
    FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T,
)

print(
    "Reference contract SHA-256        :",
    FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T,
)


if FD004_ALL_MAGNITUDE_SCALES_USABLE_630T:

    print(
        "\nPASS — the exact historical pooled-within-regime "
        "corruption reference transfers directly to all "
        "21 FD004 sensors. No corruption has yet been executed."
    )

else:

    print(
        "\nPASS WITH STRUCTURAL FINDING — the exact historical "
        "reference has been derived and locked, but one or more "
        "FD004 sensors do not have a usable pooled within-regime "
        "magnitude scale. Do NOT execute corruption yet. "
        "The next step must prospectively resolve this FIT-only "
        "structural case without using DEV outcomes."
    )

In [ ]:
# STEP 6.30U
# FD004 FIT-ONLY CORRUPTION-SCENARIO ADMISSIBILITY RESOLUTION
# + EXECUTION-MANIFEST FREEZE
#
# PURPOSE:
#   Preserve the already-frozen 441-row corruption manifest unchanged,
#   but prospectively resolve the FD004 structural scale issue using
#   FIT-only evidence before any corruption execution.
#
# RULE:
#   - bias/noise/drift require pooled_within_regime_std > 1e-12
#   - dropout/stuck do not require that magnitude scale
#
# NO corruption execution.
# NO inference / MC.
# NO DEV labels/errors.
# NO threshold/policy.
# NO TEST / official RUL.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630t", False)):
    raise RuntimeError(
        "STEP 6.30T must pass before STEP 6.30U."
    )

required_630u = [
    "FD004_CORRUPTION_MANIFEST_630S",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T",
    "FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T",
    "FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T",
    "FD004_ZERO_POOLED_SCALE_SENSORS_630T",
    "FD004_CORRUPTION_SCALE_EPSILON_630T",
]

missing_630u = [
    name_630u
    for name_630u in required_630u
    if name_630u not in globals()
]

if missing_630u:
    raise RuntimeError(
        "Missing STEP 6.30U authority objects: "
        + str(missing_630u)
    )


if bool(globals().get("FD004_CORRUPTION_EXECUTED_630T", True)):
    raise RuntimeError(
        "Corruption was unexpectedly executed before "
        "admissibility resolution."
    )

if bool(globals().get("FD004_DEV_OUTCOMES_USED_630T", True)):
    raise RuntimeError(
        "DEV outcomes were unexpectedly used before "
        "admissibility resolution."
    )

if bool(globals().get("FD004_THRESHOLD_SELECTED_630T", True)):
    raise RuntimeError(
        "Threshold was unexpectedly selected before "
        "admissibility resolution."
    )

if bool(globals().get("FD004_TEST_CONTENTS_READ_630T", True)):
    raise RuntimeError(
        "TEST was unexpectedly read before "
        "admissibility resolution."
    )

if bool(globals().get("FD004_RUL_CONTENTS_READ_630T", True)):
    raise RuntimeError(
        "Official RUL was unexpectedly read before "
        "admissibility resolution."
    )


# =============================================================================
# 1) VERIFY FROZEN MANIFEST STRUCTURE
# =============================================================================

manifest_630u = (
    FD004_CORRUPTION_MANIFEST_630S
    .copy(deep=True)
    .reset_index(drop=True)
)

expected_cols_630u = [
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
]

if list(manifest_630u.columns) != expected_cols_630u:
    raise RuntimeError(
        "Frozen STEP 6.30S manifest columns changed."
    )

if len(manifest_630u) != 441:
    raise RuntimeError(
        "Frozen STEP 6.30S manifest no longer contains "
        "exactly 441 scenarios."
    )

if manifest_630u["scenario_number"].tolist() != list(
    range(1, 442)
):
    raise RuntimeError(
        "Frozen scenario numbering is no longer exactly 1..441."
    )


# =============================================================================
# 2) BUILD FIT-ONLY SCALE LOOKUP
# =============================================================================

scale_ref_630u = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T
    .copy(deep=True)
    .reset_index(drop=True)
)

required_scale_cols_630u = [
    "sensor",
    "pooled_within_regime_std",
    "usable_gt_1e_12",
]

for col_630u in required_scale_cols_630u:
    if col_630u not in scale_ref_630u.columns:
        raise RuntimeError(
            f"Missing scale-reference column: {col_630u}"
        )

if scale_ref_630u["sensor"].duplicated().any():
    raise RuntimeError(
        "Corruption-scale reference has duplicate sensors."
    )

scale_lookup_630u = {
    str(row_630u["sensor"]):
        float(row_630u["pooled_within_regime_std"])
    for _, row_630u in scale_ref_630u.iterrows()
}

usable_lookup_630u = {
    str(row_630u["sensor"]):
        bool(row_630u["usable_gt_1e_12"])
    for _, row_630u in scale_ref_630u.iterrows()
}


manifest_sensor_set_630u = set(
    manifest_630u["sensor"].astype(str)
)

reference_sensor_set_630u = set(
    scale_lookup_630u
)

if manifest_sensor_set_630u != reference_sensor_set_630u:
    raise RuntimeError(
        "Manifest sensor set and corruption-scale reference "
        "sensor set differ."
    )


# =============================================================================
# 3) PROSPECTIVE ADMISSIBILITY RULE
# =============================================================================

FD004_MAGNITUDE_SCALE_REQUIRED_FAMILIES_630U = (
    "bias",
    "noise",
    "drift",
)

FD004_SCALE_INDEPENDENT_FAMILIES_630U = (
    "dropout",
    "stuck",
)

FD004_CORRUPTION_ADMISSIBILITY_EPSILON_630U = float(
    FD004_CORRUPTION_SCALE_EPSILON_630T
)

if FD004_CORRUPTION_ADMISSIBILITY_EPSILON_630U != 1e-12:
    raise RuntimeError(
        "Unexpected corruption-scale viability epsilon."
    )


known_families_630u = set(
    FD004_MAGNITUDE_SCALE_REQUIRED_FAMILIES_630U
    +
    FD004_SCALE_INDEPENDENT_FAMILIES_630U
)

manifest_families_630u = set(
    manifest_630u["family"].astype(str)
)

if manifest_families_630u != known_families_630u:
    raise RuntimeError(
        "Frozen manifest family set is inconsistent with "
        "the admissibility contract."
    )


admissible_630u = []
reason_630u = []
scale_value_630u = []
scale_required_630u = []


for _, row_630u in manifest_630u.iterrows():

    family_630u = str(
        row_630u["family"]
    )

    sensor_630u = str(
        row_630u["sensor"]
    )

    current_scale_630u = float(
        scale_lookup_630u[
            sensor_630u
        ]
    )

    requires_scale_630u = bool(
        family_630u
        in
        FD004_MAGNITUDE_SCALE_REQUIRED_FAMILIES_630U
    )

    scale_value_630u.append(
        current_scale_630u
    )

    scale_required_630u.append(
        requires_scale_630u
    )


    if not requires_scale_630u:

        admissible_630u.append(
            True
        )

        reason_630u.append(
            "admissible_scale_independent_family"
        )

        continue


    usable_630u = bool(
        usable_lookup_630u[
            sensor_630u
        ]
    )


    if usable_630u:

        admissible_630u.append(
            True
        )

        reason_630u.append(
            "admissible_positive_pooled_within_regime_scale"
        )

    else:

        admissible_630u.append(
            False
        )

        if current_scale_630u == 0.0:

            reason_630u.append(
                "inadmissible_zero_pooled_within_regime_scale"
            )

        else:

            reason_630u.append(
                "inadmissible_effectively_zero_pooled_within_regime_scale"
            )


# =============================================================================
# 4) FREEZE ANNOTATED MASTER MANIFEST
#
# Original 441-row STEP 6.30S object is NOT modified.
# =============================================================================

FD004_CORRUPTION_MANIFEST_ANNOTATED_630U = (
    manifest_630u
    .copy(deep=True)
)

FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
    "magnitude_scale_required"
] = np.asarray(
    scale_required_630u,
    dtype=bool,
)

FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
    "pooled_within_regime_std"
] = np.asarray(
    scale_value_630u,
    dtype=np.float64,
)

FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
    "scenario_admissible"
] = np.asarray(
    admissible_630u,
    dtype=bool,
)

FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
    "admissibility_reason"
] = np.asarray(
    reason_630u,
    dtype=object,
)


# =============================================================================
# 5) FREEZE EXECUTION MANIFEST
# =============================================================================

FD004_CORRUPTION_EXECUTION_MANIFEST_630U = (
    FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
        FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
            "scenario_admissible"
        ]
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)


FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U = (
    FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
        ~FD004_CORRUPTION_MANIFEST_ANNOTATED_630U[
            "scenario_admissible"
        ]
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)


# =============================================================================
# 6) STRUCTURAL AUDITS
# =============================================================================

nonusable_sensors_630u = tuple(
    str(x_630u)
    for x_630u
    in FD004_NONUSABLE_POOLED_SCALE_SENSORS_630T
)

zero_sensors_630u = tuple(
    str(x_630u)
    for x_630u
    in FD004_ZERO_POOLED_SCALE_SENSORS_630T
)


expected_nonusable_630u = {
    "sensor_1",
    "sensor_5",
    "sensor_18",
    "sensor_19",
}

if set(nonusable_sensors_630u) != expected_nonusable_630u:
    raise RuntimeError(
        "Nonusable sensor set changed unexpectedly."
    )

if set(zero_sensors_630u) != {
    "sensor_18"
}:
    raise RuntimeError(
        "Exactly-zero sensor set changed unexpectedly."
    )


expected_inadmissible_count_630u = int(
    len(
        expected_nonusable_630u
    )
    *
    (
        6
        +
        3
        +
        6
    )
)

actual_inadmissible_count_630u = int(
    len(
        FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U
    )
)

if expected_inadmissible_count_630u != 60:
    raise RuntimeError(
        "Internal expected inadmissible count is not 60."
    )

if actual_inadmissible_count_630u != 60:
    raise RuntimeError(
        "FD004 admissibility resolution did not produce "
        "the expected 60 structurally inadmissible scenarios."
    )


actual_admissible_count_630u = int(
    len(
        FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    )
)

if actual_admissible_count_630u != 381:
    raise RuntimeError(
        "FD004 execution manifest did not produce "
        "the expected 381 admissible scenarios."
    )


family_summary_630u = (
    FD004_CORRUPTION_MANIFEST_ANNOTATED_630U
    .groupby(
        "family",
        sort=False,
    )[
        "scenario_admissible"
    ]
    .agg(
        [
            "count",
            "sum",
        ]
    )
    .rename(
        columns={
            "count":
                "manifest_scenarios",
            "sum":
                "admissible_scenarios",
        }
    )
)

family_summary_630u[
    "inadmissible_scenarios"
] = (
    family_summary_630u[
        "manifest_scenarios"
    ]
    -
    family_summary_630u[
        "admissible_scenarios"
    ]
)


expected_family_counts_630u = {
    "bias":
        (
            126,
            102,
            24,
        ),

    "noise":
        (
            63,
            51,
            12,
        ),

    "drift":
        (
            126,
            102,
            24,
        ),

    "dropout":
        (
            63,
            63,
            0,
        ),

    "stuck":
        (
            63,
            63,
            0,
        ),
}


for family_630u, expected_630u in expected_family_counts_630u.items():

    if family_630u not in family_summary_630u.index:
        raise RuntimeError(
            f"Missing family in summary: {family_630u}"
        )

    observed_630u = tuple(
        int(x_630u)
        for x_630u in family_summary_630u.loc[
            family_630u,
            [
                "manifest_scenarios",
                "admissible_scenarios",
                "inadmissible_scenarios",
            ],
        ].tolist()
    )

    if observed_630u != expected_630u:
        raise RuntimeError(
            "Unexpected admissibility counts for "
            f"{family_630u}: {observed_630u}"
        )


inadmissible_sensor_family_630u = (
    FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U
    .groupby(
        [
            "sensor",
            "family",
        ],
        sort=False,
    )
    .size()
    .rename(
        "scenario_count"
    )
    .reset_index()
)


# =============================================================================
# 7) FINGERPRINT HELPERS
# =============================================================================

def sha256_dataframe_630u(
    df_630u
):

    h_630u = hashlib.sha256()

    h_630u.update(
        json.dumps(
            list(
                df_630u.columns
            ),
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )

    row_hashes_630u = (
        pd.util.hash_pandas_object(
            df_630u,
            index=False,
        )
        .to_numpy(
            dtype=np.uint64
        )
    )

    h_630u.update(
        np.ascontiguousarray(
            row_hashes_630u
        ).tobytes()
    )

    return h_630u.hexdigest()


FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U = (
    sha256_dataframe_630u(
        FD004_CORRUPTION_MANIFEST_ANNOTATED_630U
    )
)

FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U = (
    sha256_dataframe_630u(
        FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    )
)

FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U = (
    sha256_dataframe_630u(
        FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U
    )
)


# =============================================================================
# 8) FREEZE ADMISSIBILITY CONTRACT
# =============================================================================

FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_630U = {
    "dataset":
        "FD004",

    "frozen_manifest_rows_preserved":
        441,

    "magnitude_scale_required_families":
        list(
            FD004_MAGNITUDE_SCALE_REQUIRED_FAMILIES_630U
        ),

    "scale_independent_families":
        list(
            FD004_SCALE_INDEPENDENT_FAMILIES_630U
        ),

    "magnitude_scale_rule":
        "pooled_within_regime_std_gt_1e-12",

    "epsilon":
        float(
            FD004_CORRUPTION_ADMISSIBILITY_EPSILON_630U
        ),

    "fallback_scale_used":
        False,

    "global_raw_std_substitution_used":
        False,

    "manifest_rewritten":
        False,

    "nonusable_scale_sensors":
        list(
            nonusable_sensors_630u
        ),

    "exactly_zero_scale_sensors":
        list(
            zero_sensors_630u
        ),

    "admissible_scenarios":
        actual_admissible_count_630u,

    "inadmissible_scenarios":
        actual_inadmissible_count_630u,

    "DEV_outcomes_used":
        False,

    "corruption_executed":
        False,

    "threshold_selected":
        False,

    "TEST_read":
        False,

    "official_RUL_read":
        False,

    "corruption_scale_reference_sha256":
        str(
            FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T
        ),

    "corruption_reference_contract_sha256":
        str(
            FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T
        ),
}


FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_SHA256_630U = (
    hashlib.sha256(
        json.dumps(
            FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_630U,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# =============================================================================
# 9) STATE FLAGS
# =============================================================================

FD004_CORRUPTION_ADMISSIBILITY_RESOLVED_630U = True
FD004_CORRUPTION_EXECUTION_MANIFEST_LOCKED_630U = True

FD004_CORRUPTION_EXECUTED_630U = False
FD004_NEW_MODEL_INFERENCE_PERFORMED_630U = False
FD004_NEW_MC_EXECUTION_PERFORMED_630U = False
FD004_DEV_OUTCOMES_USED_630U = False
FD004_THRESHOLD_SELECTED_630U = False
FD004_POLICY_SELECTED_630U = False
FD004_TEST_CONTENTS_READ_630U = False
FD004_RUL_CONTENTS_READ_630U = False

all_checks_passed_630u = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30U — FD004 FIT-ONLY CORRUPTION-SCENARIO "
    "ADMISSIBILITY RESOLUTION + EXECUTION-MANIFEST FREEZE"
)

print("=" * 174)


print(
    "Original frozen manifest preserved :",
    True,
)

print(
    "Original frozen manifest rows      :",
    len(
        FD004_CORRUPTION_MANIFEST_630S
    ),
)

print(
    "Fallback scale introduced          :",
    False,
)

print(
    "Global STD substituted             :",
    False,
)

print(
    "DEV targets/errors used            :",
    False,
)

print(
    "Corruption executed                :",
    False,
)

print(
    "New model inference performed      :",
    False,
)

print(
    "New MC execution performed         :",
    False,
)

print(
    "Threshold selected                 :",
    False,
)

print(
    "Policy selected                    :",
    False,
)

print(
    "TEST contents read                 :",
    False,
)

print(
    "Official RUL contents read         :",
    False,
)


print(
    "\n--- PROSPECTIVE ADMISSIBILITY RULE ---"
)

print(
    "Scale-required families :",
    list(
        FD004_MAGNITUDE_SCALE_REQUIRED_FAMILIES_630U
    ),
)

print(
    "Scale-independent families:",
    list(
        FD004_SCALE_INDEPENDENT_FAMILIES_630U
    ),
)

print(
    "Required magnitude condition:",
    "pooled_within_regime_std > 1e-12",
)

print(
    "Nonusable scale sensors   :",
    list(
        nonusable_sensors_630u
    ),
)

print(
    "Exactly zero-scale sensors:",
    list(
        zero_sensors_630u
    ),
)


print(
    "\n--- SCENARIO COUNTS ---"
)

print(
    "Frozen master scenarios    :",
    len(
        FD004_CORRUPTION_MANIFEST_ANNOTATED_630U
    ),
)

print(
    "Admissible for execution   :",
    actual_admissible_count_630u,
)

print(
    "Structurally inadmissible  :",
    actual_inadmissible_count_630u,
)


print(
    "\nFamily-level admissibility:"
)

print(
    family_summary_630u.to_string()
)


print(
    "\nInadmissible sensor × family counts:"
)

print(
    inadmissible_sensor_family_630u.to_string(
        index=False
    )
)


print(
    "\n--- EVIDENCE FINGERPRINTS ---"
)

print(
    "Annotated manifest SHA-256   :",
    FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U,
)

print(
    "Execution manifest SHA-256   :",
    FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U,
)

print(
    "Inadmissible manifest SHA-256:",
    FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U,
)

print(
    "Admissibility contract SHA-256:",
    FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_SHA256_630U,
)


print(
    "\nPASS — the FD004 structural scale issue is prospectively "
    "resolved using FIT-only evidence without changing the frozen "
    "441-row scientific manifest. The 381 historically well-defined "
    "scenarios are now separately locked for execution; the 60 "
    "magnitude-based scenarios involving effectively zero within-regime "
    "sensor scale remain explicitly recorded as structurally "
    "inadmissible rather than being silently rescaled."
)

In [ ]:
# STEP 6.30V-0
# FD004 PRE-EXECUTION HISTORICAL CORRUPTION-OPERATOR BODY RECOVERY
#
# SOURCE-ONLY audit. Nothing is executed from the historical notebook.
# No corruption, inference, MC, DEV outcomes, threshold, TEST, or RUL.

import ast
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630u", False)):
    raise RuntimeError(
        "STEP 6.30U must pass before STEP 6.30V-0."
    )

for flag_name_630v0 in (
    "FD004_CORRUPTION_EXECUTED_630U",
    "FD004_DEV_OUTCOMES_USED_630U",
    "FD004_THRESHOLD_SELECTED_630U",
    "FD004_TEST_CONTENTS_READ_630U",
    "FD004_RUL_CONTENTS_READ_630U",
):
    if bool(globals().get(flag_name_630v0, True)):
        raise RuntimeError(
            f"Unexpected pre-execution state: "
            f"{flag_name_630v0}=True."
        )


# =============================================================================
# 1) HISTORICAL NOTEBOOK AUTHORITY
# =============================================================================

FD002_NOTEBOOK_PATH_630V0 = Path(
    r"C:\RUL_Reliability_Research\03_notebooks\07_fd002_replication.ipynb"
)

if not FD002_NOTEBOOK_PATH_630V0.is_file():
    raise RuntimeError(
        "Historical FD002 notebook not found at the frozen source path."
    )

with FD002_NOTEBOOK_PATH_630V0.open(
    "r",
    encoding="utf-8",
) as f_630v0:
    nb_630v0 = json.load(
        f_630v0
    )


# =============================================================================
# 2) RECOVER ONLY THE SIX KNOWN HISTORICAL FUNCTIONS
# =============================================================================

target_functions_630v0 = (
    "apply_fd002_bias_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_stuck_628gc",
    "apply_manifest_scenario_628ha",
)

recovered_functions_630v0 = {}


for (
    cell_index_630v0,
    cell_630v0,
) in enumerate(
    nb_630v0.get(
        "cells",
        [],
    )
):

    if (
        cell_630v0.get(
            "cell_type"
        )
        !=
        "code"
    ):
        continue


    source_630v0 = "".join(
        cell_630v0.get(
            "source",
            [],
        )
    )


    try:
        tree_630v0 = ast.parse(
            source_630v0
        )

    except SyntaxError:
        continue


    lines_630v0 = (
        source_630v0
        .splitlines()
    )


    for node_630v0 in tree_630v0.body:

        if not isinstance(
            node_630v0,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            continue


        if (
            node_630v0.name
            not in
            target_functions_630v0
        ):
            continue


        if not hasattr(
            node_630v0,
            "end_lineno",
        ):
            raise RuntimeError(
                "Python AST does not expose end_lineno; "
                "exact source recovery stopped."
            )


        fn_source_630v0 = "\n".join(
            lines_630v0[
                int(
                    node_630v0.lineno
                ) - 1:
                int(
                    node_630v0.end_lineno
                )
            ]
        )


        if (
            node_630v0.name
            in
            recovered_functions_630v0
        ):
            raise RuntimeError(
                "Duplicate historical definition found for "
                f"{node_630v0.name}."
            )


        recovered_functions_630v0[
            node_630v0.name
        ] = {
            "cell_index":
                int(
                    cell_index_630v0
                ),

            "source":
                fn_source_630v0,

            "source_sha256":
                hashlib.sha256(
                    fn_source_630v0.encode(
                        "utf-8"
                    )
                ).hexdigest(),
        }


# =============================================================================
# 3) REQUIRE COMPLETE EXACT RECOVERY
# =============================================================================

missing_functions_630v0 = [
    fn_630v0
    for fn_630v0
    in target_functions_630v0
    if fn_630v0
    not in recovered_functions_630v0
]

if missing_functions_630v0:
    raise RuntimeError(
        "Missing exact historical corruption functions: "
        + str(
            missing_functions_630v0
        )
    )


# =============================================================================
# 4) LOCK SOURCE-RECOVERY METADATA
# =============================================================================

FD004_HISTORICAL_CORRUPTION_FUNCTION_BODIES_630V0 = tuple(
    {
        "name":
            fn_630v0,

        "cell_index":
            recovered_functions_630v0[
                fn_630v0
            ][
                "cell_index"
            ],

        "source_sha256":
            recovered_functions_630v0[
                fn_630v0
            ][
                "source_sha256"
            ],
    }

    for fn_630v0
    in target_functions_630v0
)


FD004_HISTORICAL_CORRUPTION_FUNCTION_RECOVERY_LOCKED_630V0 = True

all_checks_passed_630v0 = True


# =============================================================================
# 5) REPORT
# =============================================================================

print(
    "=" * 174
)

print(
    "STEP 6.30V-0 — FD004 PRE-EXECUTION HISTORICAL "
    "CORRUPTION-OPERATOR BODY RECOVERY"
)

print(
    "=" * 174
)


print(
    "Historical source read            :",
    True,
)

print(
    "Historical code executed          :",
    False,
)

print(
    "Corruption executed               :",
    False,
)

print(
    "New model inference performed     :",
    False,
)

print(
    "New MC execution performed        :",
    False,
)

print(
    "DEV targets/errors used           :",
    False,
)

print(
    "Threshold selected                :",
    False,
)

print(
    "Policy selected                   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


for fn_630v0 in target_functions_630v0:

    item_630v0 = (
        recovered_functions_630v0[
            fn_630v0
        ]
    )

    print(
        "\n"
        +
        "-" * 174
    )

    print(
        "Function      :",
        fn_630v0,
    )

    print(
        "Cell index    :",
        item_630v0[
            "cell_index"
        ],
    )

    print(
        "Body SHA-256  :",
        item_630v0[
            "source_sha256"
        ],
    )

    print(
        "-" * 174
    )

    print(
        item_630v0[
            "source"
        ]
    )


print(
    "\nPASS — the six exact historical FD002 corruption/dispatch "
    "function bodies were recovered as source text only. "
    "Nothing was executed."
)

In [ ]:
# STEP 6.30V-1
# FD004 CORRUPTION->MODEL SMOKE-BRIDGE AUTHORITY LOCATOR
#
# Metadata only:
# - no corruption
# - no inference / MC
# - no DEV target/error values
# - no threshold/policy
# - no TEST / RUL

import numpy as np
import pandas as pd

if not bool(globals().get("all_checks_passed_630u", False)):
    raise RuntimeError("STEP 6.30U must pass before STEP 6.30V-1.")

if not bool(
    globals().get(
        "FD004_HISTORICAL_CORRUPTION_FUNCTION_RECOVERY_LOCKED_630V0",
        False,
    )
):
    raise RuntimeError("STEP 6.30V-0 must pass before STEP 6.30V-1.")

for flag_name_630v1 in (
    "FD004_CORRUPTION_EXECUTED_630U",
    "FD004_DEV_OUTCOMES_USED_630U",
    "FD004_THRESHOLD_SELECTED_630U",
    "FD004_TEST_CONTENTS_READ_630U",
    "FD004_RUL_CONTENTS_READ_630U",
):
    if bool(globals().get(flag_name_630v1, True)):
        raise RuntimeError(
            f"Unexpected pre-smoke state: {flag_name_630v1}=True."
        )

candidate_names_630v1 = (
    # Preprocessing / windows
    "FD004_DEV_NORMALIZED_630F",
    "X_DEV_FD004_630G",
    "FD004_X_DEV_630G",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "FD004_DEV_WINDOW_UNIT_IDS_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "FD004_DEV_WINDOW_ENDPOINT_CYCLES_630G",

    # Persistent model / device
    "FD004_MODEL_630I",
    "FD004_MODEL_FD004_630I",
    "FD004_GRU_MODEL_630I",
    "model_fd004_630i",
    "model_630i",
    "FD004_DEVICE_630I",
    "device_fd004_630i",
    "device_630i",

    # Trained/restored model fingerprints
    "FD004_RESTORED_TRAINED_MODEL_SHA256_630J",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_RESTORED_MODEL_SHA256_630J",
    "restored_trained_model_sha256_630j",
    "trained_model_sha256_630j",
    "model_sha256_after_restore_630j",

    # Retained clean DEV prediction
    "FD004_BEST_DEV_PREDICTION_630J",
    "FD004_BEST_DEV_PRED_630J",
    "best_dev_prediction_630j",
    "best_dev_pred_630j",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_PRED_630K",
    "clean_dev_prediction_630k",
    "clean_dev_pred_630k",

    # Frozen model-feature authorities
    "FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2",
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "FD004_MODEL_OP_COLS_630E",
    "FD004_SENSOR_COLS_630E",
    "FD004_FIT_REGIME_SENSOR_MEANS_630F",
    "FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F",
    "FD004_MODEL_OP_SCALER_630F",
)

rows_630v1 = []

for name_630v1 in candidate_names_630v1:
    present_630v1 = name_630v1 in globals()

    if not present_630v1:
        rows_630v1.append(
            {
                "name": name_630v1,
                "present": False,
                "type": None,
                "shape_or_len": None,
                "dtype_or_value": None,
            }
        )
        continue

    obj_630v1 = globals()[name_630v1]
    type_name_630v1 = type(obj_630v1).__name__

    shape_or_len_630v1 = None
    dtype_or_value_630v1 = None

    if isinstance(obj_630v1, np.ndarray):
        shape_or_len_630v1 = tuple(obj_630v1.shape)
        dtype_or_value_630v1 = str(obj_630v1.dtype)

    elif isinstance(obj_630v1, pd.DataFrame):
        shape_or_len_630v1 = tuple(obj_630v1.shape)
        dtype_or_value_630v1 = "DataFrame"

    elif isinstance(obj_630v1, (tuple, list)):
        shape_or_len_630v1 = len(obj_630v1)
        dtype_or_value_630v1 = type_name_630v1

    elif isinstance(obj_630v1, str):
        shape_or_len_630v1 = len(obj_630v1)
        dtype_or_value_630v1 = obj_630v1

    else:
        try:
            import torch

            if isinstance(obj_630v1, torch.device):
                dtype_or_value_630v1 = str(obj_630v1)

            elif isinstance(obj_630v1, torch.nn.Module):
                dtype_or_value_630v1 = obj_630v1.__class__.__name__

            else:
                dtype_or_value_630v1 = type_name_630v1

        except Exception:
            dtype_or_value_630v1 = type_name_630v1

    rows_630v1.append(
        {
            "name": name_630v1,
            "present": True,
            "type": type_name_630v1,
            "shape_or_len": shape_or_len_630v1,
            "dtype_or_value": dtype_or_value_630v1,
        }
    )

locator_df_630v1 = pd.DataFrame(rows_630v1)

# Names-only fallback for the uncertain 630I/630J/630K namespaces.
# No object values are read here.
fallback_names_630v1 = sorted(
    name_630v1
    for name_630v1 in globals()
    if (
        "FD004" in name_630v1.upper()
        and any(
            tag_630v1 in name_630v1.upper()
            for tag_630v1 in ("630I", "630J", "630K")
        )
        and any(
            token_630v1 in name_630v1.upper()
            for token_630v1 in (
                "MODEL",
                "DEVICE",
                "PRED",
                "SHA256",
            )
        )
    )
)

FD004_SMOKE_BRIDGE_LOCATOR_COMPLETED_630V1 = True
all_checks_passed_630v1 = True

print("=" * 174)
print(
    "STEP 6.30V-1 — FD004 CORRUPTION->MODEL "
    "SMOKE-BRIDGE AUTHORITY LOCATOR"
)
print("=" * 174)

print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV target/error values inspected :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- EXPLICITLY WHITELISTED AUTHORITY CANDIDATES ---")
print(locator_df_630v1.to_string(index=False))

print(
    "\n--- NAMES-ONLY FALLBACK: "
    "FD004 630I/630J/630K MODEL/PRED AUTHORITY ---"
)

for name_630v1 in fallback_names_630v1:
    print(name_630v1)

print(
    "\nPASS — only whitelisted object metadata plus names-only "
    "630I/630J/630K authority discovery was performed. "
    "No numerical DEV outcome, corruption, inference, MC, "
    "TEST, or RUL operation occurred."
)

In [ ]:
# STEP 6.30V-2
# FD004 SINGLE-ADMISSIBLE-SCENARIO
# CORRUPTION -> PREPROCESSING -> WINDOW -> MODEL SMOKE TEST

import numpy as np
import pandas as pd
import torch
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v1", False)):
    raise RuntimeError(
        "STEP 6.30V-1 must pass before STEP 6.30V-2."
    )


required_630v2 = [
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_MODEL_OP_SCALER_630F",
    "FD004_FIT_REGIME_SENSOR_MEANS_630F",
    "FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F",
    "FD004_MODEL_OP_COLS_630E",
    "FD004_SENSOR_COLS_630E",
    "FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2",
    "FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2",
    "X_DEV_FD004_630G",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "FD004_DEV_NORMALIZED_630F",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_CLEAN_DEV_PREDICTION_630K",
]

missing_630v2 = [
    name_630v2
    for name_630v2 in required_630v2
    if name_630v2 not in globals()
]

if missing_630v2:
    raise RuntimeError(
        "Missing STEP 6.30V-2 authorities: "
        + str(missing_630v2)
    )


for flag_name_630v2 in (
    "FD004_DEV_OUTCOMES_USED_630U",
    "FD004_THRESHOLD_SELECTED_630U",
    "FD004_TEST_CONTENTS_READ_630U",
    "FD004_RUL_CONTENTS_READ_630U",
):
    if bool(globals().get(flag_name_630v2, True)):
        raise RuntimeError(
            f"Unexpected pre-smoke state: "
            f"{flag_name_630v2}=True."
        )


# =============================================================================
# 1) HELPERS
# =============================================================================

def sha256_array_630v2(arr_630v2):

    arr_630v2 = np.ascontiguousarray(
        np.asarray(arr_630v2)
    )

    h_630v2 = hashlib.sha256()

    h_630v2.update(
        str(
            arr_630v2.dtype
        ).encode("utf-8")
    )

    h_630v2.update(
        json.dumps(
            list(
                arr_630v2.shape
            ),
            separators=(",", ":"),
        ).encode("utf-8")
    )

    h_630v2.update(
        arr_630v2.tobytes()
    )

    return h_630v2.hexdigest()


def sha256_model_state_630v2(
    model_630v2
):

    h_630v2 = hashlib.sha256()

    for (
        key_630v2,
        tensor_630v2,
    ) in model_630v2.state_dict().items():

        arr_630v2 = np.ascontiguousarray(
            tensor_630v2
            .detach()
            .cpu()
            .numpy()
        )

        h_630v2.update(
            key_630v2.encode(
                "utf-8"
            )
        )

        h_630v2.update(
            str(
                arr_630v2.dtype
            ).encode(
                "utf-8"
            )
        )

        h_630v2.update(
            json.dumps(
                list(
                    arr_630v2.shape
                ),
                separators=(",", ":"),
            ).encode(
                "utf-8"
            )
        )

        h_630v2.update(
            arr_630v2.tobytes()
        )

    return h_630v2.hexdigest()


def normalize_fd004_raw_630v2(
    raw_df_630v2,
    regime_ids_630v2,
):

    raw_local_630v2 = (
        raw_df_630v2
        .reset_index(
            drop=True
        )
        .copy()
    )

    regime_ids_630v2 = np.asarray(
        regime_ids_630v2,
        dtype=np.int64,
    ).reshape(-1)


    if (
        len(
            raw_local_630v2
        )
        !=
        regime_ids_630v2.size
    ):
        raise RuntimeError(
            "Raw rows and regime IDs are misaligned."
        )


    op_cols_630v2 = tuple(
        FD004_MODEL_OP_COLS_630E
    )

    sensor_cols_630v2 = tuple(
        FD004_SENSOR_COLS_630E
    )

    raw_features_630v2 = tuple(
        FD004_AUTHORITATIVE_RAW_MODEL_FEATURES_630ER2
    )

    norm_features_630v2 = tuple(
        FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
    )


    if (
        raw_features_630v2
        !=
        op_cols_630v2
        +
        sensor_cols_630v2
    ):
        raise RuntimeError(
            "Frozen raw model-feature order is inconsistent."
        )


    expected_norm_630v2 = tuple(
        f"{col_630v2}_z"
        for col_630v2
        in raw_features_630v2
    )


    if (
        norm_features_630v2
        !=
        expected_norm_630v2
    ):
        raise RuntimeError(
            "Frozen normalized model-feature order "
            "is inconsistent."
        )


    op_z_630v2 = np.asarray(
        FD004_MODEL_OP_SCALER_630F.transform(
            raw_local_630v2.loc[
                :,
                list(
                    op_cols_630v2
                ),
            ]
            .to_numpy(
                dtype=np.float64
            )
        ),
        dtype=np.float64,
    )


    sensor_raw_630v2 = (
        raw_local_630v2.loc[
            :,
            list(
                sensor_cols_630v2
            ),
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    sensor_z_630v2 = np.empty_like(
        sensor_raw_630v2,
        dtype=np.float64,
    )


    means_630v2 = np.asarray(
        FD004_FIT_REGIME_SENSOR_MEANS_630F,
        dtype=np.float64,
    )

    stds_630v2 = np.asarray(
        FD004_FIT_REGIME_SENSOR_STDS_EFFECTIVE_630F,
        dtype=np.float64,
    )


    if (
        means_630v2.shape
        !=
        (
            6,
            len(
                sensor_cols_630v2
            ),
        )
    ):
        raise RuntimeError(
            "Frozen sensor mean matrix has unexpected shape."
        )


    if (
        stds_630v2.shape
        !=
        (
            6,
            len(
                sensor_cols_630v2
            ),
        )
    ):
        raise RuntimeError(
            "Frozen sensor STD matrix has unexpected shape."
        )


    for regime_id_630v2 in range(6):

        mask_630v2 = (
            regime_ids_630v2
            ==
            regime_id_630v2
        )

        if not np.any(
            mask_630v2
        ):
            raise RuntimeError(
                f"No rows found for frozen DEV regime "
                f"{regime_id_630v2}."
            )


        sensor_z_630v2[
            mask_630v2
        ] = (
            sensor_raw_630v2[
                mask_630v2
            ]
            -
            means_630v2[
                regime_id_630v2
            ]
        ) / (
            stds_630v2[
                regime_id_630v2
            ]
        )


    model_matrix_630v2 = np.concatenate(
        [
            op_z_630v2,
            sensor_z_630v2,
        ],
        axis=1,
    )


    if not np.isfinite(
        model_matrix_630v2
    ).all():
        raise RuntimeError(
            "Non-finite normalized values produced."
        )


    result_630v2 = pd.DataFrame(
        {
            "unit_id":
                raw_local_630v2[
                    "unit_id"
                ].to_numpy(
                    dtype=np.int64
                ),

            "cycle":
                raw_local_630v2[
                    "cycle"
                ].to_numpy(
                    dtype=np.int64
                ),

            "regime_id":
                regime_ids_630v2,
        }
    )


    for (
        feature_index_630v2,
        feature_name_630v2,
    ) in enumerate(
        norm_features_630v2
    ):

        result_630v2[
            feature_name_630v2
        ] = (
            model_matrix_630v2[
                :,
                feature_index_630v2
            ]
        )


    return result_630v2


def build_windows_fd004_630v2(
    norm_df_630v2
):

    feature_cols_630v2 = tuple(
        FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
    )

    windows_630v2 = []
    unit_ids_630v2 = []
    endpoint_cycles_630v2 = []


    for (
        unit_id_630v2,
        group_630v2,
    ) in norm_df_630v2.groupby(
        "unit_id",
        sort=False,
    ):

        group_630v2 = (
            group_630v2
            .sort_values(
                "cycle"
            )
        )


        values_630v2 = (
            group_630v2.loc[
                :,
                list(
                    feature_cols_630v2
                ),
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        cycles_630v2 = (
            group_630v2[
                "cycle"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        if len(
            group_630v2
        ) < 30:
            raise RuntimeError(
                f"Engine {unit_id_630v2} "
                "has fewer than 30 DEV rows."
            )


        for end_index_630v2 in range(
            29,
            len(
                group_630v2
            ),
        ):

            windows_630v2.append(
                values_630v2[
                    end_index_630v2 - 29:
                    end_index_630v2 + 1
                ]
            )

            unit_ids_630v2.append(
                int(
                    unit_id_630v2
                )
            )

            endpoint_cycles_630v2.append(
                int(
                    cycles_630v2[
                        end_index_630v2
                    ]
                )
            )


    X_630v2 = np.stack(
        windows_630v2,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    return (
        X_630v2,

        np.asarray(
            unit_ids_630v2,
            dtype=np.int64,
        ),

        np.asarray(
            endpoint_cycles_630v2,
            dtype=np.int64,
        ),
    )


# =============================================================================
# 2) RECONSTRUCT CLEAN PREPROCESSING BRIDGE — NO INFERENCE
# =============================================================================

clean_rebuilt_norm_630v2 = (
    normalize_fd004_raw_630v2(
        FD004_DEV_RAW_630C,
        FD004_DEV_REGIME_IDS_630F,
    )
)


clean_authority_matrix_630v2 = (
    FD004_DEV_NORMALIZED_630F.loc[
        :,
        list(
            FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
        ),
    ]
    .to_numpy(
        dtype=np.float64
    )
)


clean_rebuilt_matrix_630v2 = (
    clean_rebuilt_norm_630v2.loc[
        :,
        list(
            FD004_AUTHORITATIVE_MODEL_FEATURES_630ER2
        ),
    ]
    .to_numpy(
        dtype=np.float64
    )
)


clean_norm_max_abs_diff_630v2 = float(
    np.max(
        np.abs(
            clean_rebuilt_matrix_630v2
            -
            clean_authority_matrix_630v2
        )
    )
)


clean_norm_exact_630v2 = bool(
    np.array_equal(
        clean_rebuilt_matrix_630v2,
        clean_authority_matrix_630v2,
    )
)


clean_norm_close_630v2 = bool(
    np.allclose(
        clean_rebuilt_matrix_630v2,
        clean_authority_matrix_630v2,
        atol=1e-10,
        rtol=1e-12,
    )
)


if not clean_norm_close_630v2:
    raise RuntimeError(
        "Rebuilt clean normalization does not reproduce "
        "frozen STEP 6.30F authority."
    )


(
    X_clean_rebuilt_630v2,
    clean_units_630v2,
    clean_cycles_630v2,
) = build_windows_fd004_630v2(
    clean_rebuilt_norm_630v2
)


if not np.array_equal(
    clean_units_630v2,
    DEV_WINDOW_UNIT_IDS_FD004_630G,
):
    raise RuntimeError(
        "Rebuilt clean DEV window-unit ordering differs "
        "from STEP 6.30G."
    )


if not np.array_equal(
    clean_cycles_630v2,
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
):
    raise RuntimeError(
        "Rebuilt clean DEV endpoint-cycle ordering differs "
        "from STEP 6.30G."
    )


clean_window_max_abs_diff_630v2 = float(
    np.max(
        np.abs(
            X_clean_rebuilt_630v2
            -
            np.asarray(
                X_DEV_FD004_630G,
                dtype=np.float32,
            )
        )
    )
)


clean_window_exact_630v2 = bool(
    np.array_equal(
        X_clean_rebuilt_630v2,
        np.asarray(
            X_DEV_FD004_630G,
            dtype=np.float32,
        ),
    )
)


if not clean_window_exact_630v2:
    raise RuntimeError(
        "Rebuilt clean DEV windows are not byte-identical "
        "to frozen STEP 6.30G windows."
    )


# =============================================================================
# 3) SELECT EXACTLY ONE ADMISSIBLE SMOKE SCENARIO
# =============================================================================

smoke_row_630v2 = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .iloc[0]
    .copy()
)


if int(
    smoke_row_630v2[
        "scenario_number"
    ]
) != 7:
    raise RuntimeError(
        "Expected earliest admissible scenario_number 7."
    )


if str(
    smoke_row_630v2[
        "family"
    ]
) != "bias":
    raise RuntimeError(
        "Expected earliest admissible smoke family "
        "to be bias."
    )


if str(
    smoke_row_630v2[
        "sensor"
    ]
) != "sensor_2":
    raise RuntimeError(
        "Expected earliest admissible smoke sensor "
        "to be sensor_2."
    )


sensor_630v2 = str(
    smoke_row_630v2[
        "sensor"
    ]
)

alpha_630v2 = float(
    smoke_row_630v2[
        "alpha"
    ]
)

direction_630v2 = int(
    smoke_row_630v2[
        "direction"
    ]
)


scale_match_630v2 = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T.loc[
        FD004_CORRUPTION_SCALE_REFERENCE_630T[
            "sensor"
        ].astype(str)
        ==
        sensor_630v2,
        "pooled_within_regime_std",
    ]
)


if len(
    scale_match_630v2
) != 1:
    raise RuntimeError(
        "Smoke sensor scale lookup is not one-to-one."
    )


scale_630v2 = float(
    scale_match_630v2.iloc[0]
)


if (
    not np.isfinite(
        scale_630v2
    )
    or
    scale_630v2 <= 1e-12
):
    raise RuntimeError(
        "Smoke scenario does not have a usable "
        "historical corruption scale."
    )


# Exact recovered historical bias semantics:
# bias = direction * alpha * pooled within-regime scale
bias_amount_630v2 = float(
    direction_630v2
    *
    alpha_630v2
    *
    scale_630v2
)


# =============================================================================
# 4) EXECUTE ONLY THAT ONE RAW CORRUPTION
# =============================================================================

FD004_SMOKE_CORRUPTED_RAW_630V2 = (
    FD004_DEV_RAW_630C
    .reset_index(
        drop=True
    )
    .copy(
        deep=True
    )
)


FD004_SMOKE_CORRUPTED_RAW_630V2[
    sensor_630v2
] = (
    FD004_SMOKE_CORRUPTED_RAW_630V2[
        sensor_630v2
    ]
    .to_numpy(
        dtype=np.float64
    )
    +
    bias_amount_630v2
)


# Verify that only the requested raw sensor changed.
for col_630v2 in FD004_DEV_RAW_630C.columns:

    before_630v2 = (
        FD004_DEV_RAW_630C
        .reset_index(
            drop=True
        )[
            col_630v2
        ]
        .to_numpy()
    )

    after_630v2 = (
        FD004_SMOKE_CORRUPTED_RAW_630V2[
            col_630v2
        ]
        .to_numpy()
    )


    if col_630v2 == sensor_630v2:

        if np.array_equal(
            before_630v2,
            after_630v2,
        ):
            raise RuntimeError(
                "Smoke corruption did not modify "
                "the target sensor."
            )

    else:

        if not np.array_equal(
            before_630v2,
            after_630v2,
        ):
            raise RuntimeError(
                "Smoke corruption unexpectedly modified "
                f"{col_630v2}."
            )


# =============================================================================
# 5) FROZEN PREPROCESSING + WINDOW RECONSTRUCTION
# =============================================================================

FD004_SMOKE_NORMALIZED_630V2 = (
    normalize_fd004_raw_630v2(
        FD004_SMOKE_CORRUPTED_RAW_630V2,
        FD004_DEV_REGIME_IDS_630F,
    )
)


(
    FD004_SMOKE_X_DEV_630V2,
    FD004_SMOKE_WINDOW_UNIT_IDS_630V2,
    FD004_SMOKE_WINDOW_ENDPOINT_CYCLES_630V2,
) = build_windows_fd004_630v2(
    FD004_SMOKE_NORMALIZED_630V2
)


if (
    FD004_SMOKE_X_DEV_630V2.shape
    !=
    np.asarray(
        X_DEV_FD004_630G
    ).shape
):
    raise RuntimeError(
        "Smoke corrupted DEV window tensor shape changed."
    )


if not np.array_equal(
    FD004_SMOKE_WINDOW_UNIT_IDS_630V2,
    DEV_WINDOW_UNIT_IDS_FD004_630G,
):
    raise RuntimeError(
        "Smoke corrupted window-unit ordering changed."
    )


if not np.array_equal(
    FD004_SMOKE_WINDOW_ENDPOINT_CYCLES_630V2,
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
):
    raise RuntimeError(
        "Smoke corrupted endpoint-cycle ordering changed."
    )


if not np.isfinite(
    FD004_SMOKE_X_DEV_630V2
).all():
    raise RuntimeError(
        "Smoke corrupted model-input tensor "
        "contains non-finite values."
    )


# =============================================================================
# 6) EXACTLY ONE DETERMINISTIC CORRUPTED-SCENARIO INFERENCE
# =============================================================================

model_630v2 = (
    MODEL_FD004_630I
)

device_630v2 = (
    FD004_DEVICE_630I
)


model_mode_before_630v2 = bool(
    model_630v2.training
)


model_hash_before_630v2 = (
    sha256_model_state_630v2(
        model_630v2
    )
)


model_630v2.eval()

pred_parts_630v2 = []


with torch.no_grad():

    for start_630v2 in range(
        0,
        len(
            FD004_SMOKE_X_DEV_630V2
        ),
        512,
    ):

        xb_630v2 = torch.from_numpy(
            FD004_SMOKE_X_DEV_630V2[
                start_630v2:
                start_630v2 + 512
            ]
        ).to(
            device_630v2
        )


        pred_630v2 = (
            model_630v2(
                xb_630v2
            )
        )


        pred_parts_630v2.append(
            pred_630v2
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
            .astype(
                np.float32,
                copy=False,
            )
        )


FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2 = (
    np.concatenate(
        pred_parts_630v2
    )
)


model_630v2.train(
    model_mode_before_630v2
)


model_hash_after_630v2 = (
    sha256_model_state_630v2(
        model_630v2
    )
)


if (
    model_hash_before_630v2
    !=
    model_hash_after_630v2
):
    raise RuntimeError(
        "Model state changed during smoke inference."
    )


if (
    bool(
        model_630v2.training
    )
    !=
    model_mode_before_630v2
):
    raise RuntimeError(
        "Model mode was not restored "
        "after smoke inference."
    )


if (
    FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2.shape
    !=
    (10824,)
):
    raise RuntimeError(
        "Unexpected smoke prediction shape."
    )


if not np.isfinite(
    FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2
).all():
    raise RuntimeError(
        "Smoke predictions contain non-finite values."
    )


# =============================================================================
# 7) LABEL-FREE CHANGE CHECK AGAINST FROZEN CLEAN PREDICTION
# =============================================================================

clean_pred_630v2 = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float32,
).reshape(-1)


if (
    clean_pred_630v2.shape
    !=
    (10824,)
):
    raise RuntimeError(
        "Frozen clean DEV prediction has "
        "unexpected shape."
    )


pred_delta_630v2 = (
    FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2
    .astype(
        np.float64
    )
    -
    clean_pred_630v2.astype(
        np.float64
    )
)


changed_prediction_count_630v2 = int(
    np.count_nonzero(
        pred_delta_630v2
        !=
        0.0
    )
)


if (
    changed_prediction_count_630v2
    ==
    0
):
    raise RuntimeError(
        "Smoke corruption produced no deterministic "
        "prediction change; bridge audit failed."
    )


# =============================================================================
# 8) LOCK SMOKE EVIDENCE
# =============================================================================

FD004_SMOKE_SCENARIO_ID_630V2 = str(
    smoke_row_630v2[
        "scenario_id"
    ]
)

FD004_SMOKE_SCENARIO_NUMBER_630V2 = int(
    smoke_row_630v2[
        "scenario_number"
    ]
)

FD004_SMOKE_BIAS_AMOUNT_630V2 = float(
    bias_amount_630v2
)


FD004_SMOKE_X_SHA256_630V2 = (
    sha256_array_630v2(
        FD004_SMOKE_X_DEV_630V2
    )
)


FD004_SMOKE_PREDICTION_SHA256_630V2 = (
    sha256_array_630v2(
        FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2
    )
)


FD004_SMOKE_BRIDGE_CONTRACT_630V2 = {
    "dataset":
        "FD004",

    "scenario_id":
        FD004_SMOKE_SCENARIO_ID_630V2,

    "scenario_number":
        FD004_SMOKE_SCENARIO_NUMBER_630V2,

    "family":
        str(
            smoke_row_630v2[
                "family"
            ]
        ),

    "sensor":
        sensor_630v2,

    "alpha":
        alpha_630v2,

    "direction":
        direction_630v2,

    "historical_scale":
        scale_630v2,

    "bias_amount":
        bias_amount_630v2,

    "windows":
        10824,

    "window_length":
        30,

    "features":
        24,

    "deterministic_inference_runs":
        1,

    "mc_runs":
        0,

    "DEV_outcomes_used":
        False,

    "threshold_selected":
        False,

    "TEST_read":
        False,

    "official_RUL_read":
        False,
}


FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2 = (
    hashlib.sha256(
        json.dumps(
            FD004_SMOKE_BRIDGE_CONTRACT_630V2,
            sort_keys=True,
            separators=(",", ":"),
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# =============================================================================
# 9) STATE FLAGS
# =============================================================================

FD004_SMOKE_CORRUPTION_EXECUTED_630V2 = True
FD004_FULL_CORRUPTION_EXECUTED_630V2 = False

FD004_NEW_MC_EXECUTION_PERFORMED_630V2 = False
FD004_DEV_OUTCOMES_USED_630V2 = False
FD004_THRESHOLD_SELECTED_630V2 = False
FD004_TEST_CONTENTS_READ_630V2 = False
FD004_RUL_CONTENTS_READ_630V2 = False

all_checks_passed_630v2 = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 174)

print(
    "STEP 6.30V-2 — FD004 SINGLE-ADMISSIBLE-SCENARIO "
    "CORRUPTION->PREPROCESSING->WINDOW->MODEL SMOKE TEST"
)

print("=" * 174)


print(
    "Smoke corruption executed         :",
    True,
)

print(
    "Full corruption run executed      :",
    False,
)

print(
    "Deterministic inference runs      :",
    1,
)

print(
    "MC execution performed            :",
    False,
)

print(
    "DEV targets/errors used           :",
    False,
)

print(
    "Threshold selected                :",
    False,
)

print(
    "Policy selected                   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print(
    "\n--- CLEAN BRIDGE RECONSTRUCTION ---"
)

print(
    "Clean normalization exact         :",
    clean_norm_exact_630v2,
)

print(
    "Clean normalization max abs diff  :",
    clean_norm_max_abs_diff_630v2,
)

print(
    "Clean window tensor exact         :",
    clean_window_exact_630v2,
)

print(
    "Clean window max abs diff         :",
    clean_window_max_abs_diff_630v2,
)

print(
    "Clean window shape                :",
    X_clean_rebuilt_630v2.shape,
)


print(
    "\n--- SMOKE SCENARIO ---"
)

print(
    "Scenario ID / number              :",
    (
        FD004_SMOKE_SCENARIO_ID_630V2,
        FD004_SMOKE_SCENARIO_NUMBER_630V2,
    ),
)

print(
    "Family / sensor                   :",
    (
        str(
            smoke_row_630v2[
                "family"
            ]
        ),
        sensor_630v2,
    ),
)

print(
    "Severity / alpha / direction      :",
    (
        str(
            smoke_row_630v2[
                "severity"
            ]
        ),
        alpha_630v2,
        direction_630v2,
    ),
)

print(
    "Historical pooled scale           :",
    scale_630v2,
)

print(
    "Applied bias amount               :",
    bias_amount_630v2,
)


print(
    "\n--- CORRUPTED MODEL INPUT / PREDICTION ---"
)

print(
    "Corrupted window shape            :",
    FD004_SMOKE_X_DEV_630V2.shape,
)

print(
    "Window membership exact           :",
    np.array_equal(
        FD004_SMOKE_WINDOW_UNIT_IDS_630V2,
        DEV_WINDOW_UNIT_IDS_FD004_630G,
    ),
)

print(
    "Endpoint-cycle alignment exact    :",
    np.array_equal(
        FD004_SMOKE_WINDOW_ENDPOINT_CYCLES_630V2,
        DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    ),
)

print(
    "Prediction shape                  :",
    FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2.shape,
)

print(
    "Changed prediction count          :",
    changed_prediction_count_630v2,
)

print(
    "Prediction delta min/median/mean/max:",
    (
        float(
            np.min(
                pred_delta_630v2
            )
        ),
        float(
            np.median(
                pred_delta_630v2
            )
        ),
        float(
            np.mean(
                pred_delta_630v2
            )
        ),
        float(
            np.max(
                pred_delta_630v2
            )
        ),
    ),
)


print(
    "\n--- MODEL SAFETY ---"
)

print(
    "Model state unchanged             :",
    (
        model_hash_before_630v2
        ==
        model_hash_after_630v2
    ),
)

print(
    "Model mode restored               :",
    (
        bool(
            model_630v2.training
        )
        ==
        model_mode_before_630v2
    ),
)

print(
    "Resident trained-model authority  :",
    FD004_TRAINED_MODEL_SHA256_630J,
)


print(
    "\n--- EVIDENCE FINGERPRINTS ---"
)

print(
    "Smoke X SHA-256                   :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "Smoke prediction SHA-256          :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)

print(
    "Smoke bridge contract SHA-256     :",
    FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2,
)


print(
    "\nPASS — exactly one admissible FD004 corruption scenario "
    "was carried through the frozen preprocessing/window/model "
    "bridge. No MC, DEV outcome scoring, threshold selection, "
    "TEST, or official RUL access occurred."
)

In [ ]:
# STEP 6.30V-3
# FD004 FULL CORRUPTION RELIABILITY-CHAIN AUTHORITY LOCATOR
#
# Metadata / names only.
# NO corruption.
# NO deterministic inference.
# NO MC execution.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import pandas as pd
import inspect


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(
    globals().get(
        "all_checks_passed_630v2",
        False,
    )
):
    raise RuntimeError(
        "STEP 6.30V-2 must pass before STEP 6.30V-3."
    )


if bool(
    globals().get(
        "FD004_FULL_CORRUPTION_EXECUTED_630V2",
        True,
    )
):
    raise RuntimeError(
        "Full corruption execution unexpectedly occurred already."
    )


for flag_name_630v3 in (
    "FD004_DEV_OUTCOMES_USED_630V2",
    "FD004_THRESHOLD_SELECTED_630V2",
    "FD004_TEST_CONTENTS_READ_630V2",
    "FD004_RUL_CONTENTS_READ_630V2",
):
    if bool(
        globals().get(
            flag_name_630v3,
            True,
        )
    ):
        raise RuntimeError(
            f"Unexpected state before reliability-chain locator: "
            f"{flag_name_630v3}=True."
        )


# =============================================================================
# 1) EXPLICIT WHITELIST
# =============================================================================

candidate_names_630v3 = (
    # -------------------------------------------------------------------------
    # Canonical MC helper / protocol
    # -------------------------------------------------------------------------
    "mc_dropout_predict_fd004_630m",
    "MC_DROPOUT_PREDICT_FD004_630M",
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_DDOF_630L",
    "FD004_MC_PROTOCOL_CONTRACT_SHA256_630L",

    # -------------------------------------------------------------------------
    # FIT calibration / phase references
    # -------------------------------------------------------------------------
    "FD004_PHASE_EDGES_630O",
    "FD004_PHASE_REFERENCE_LIST_630O",
    "FD004_PHASE_REFERENCES_630O",
    "FD004_FIT_PHASE_REFERENCE_LIST_630O",
    "FD004_FIT_PHASE_REFERENCES_630O",
    "FD004_FIT_DETERMINISTIC_PREDICTION_630O",
    "FD004_FIT_MC_STD_630O",
    "FD004_CALIBRATION_EVIDENCE_SHA256_630O",

    # -------------------------------------------------------------------------
    # Phase-adjustment / G1
    # -------------------------------------------------------------------------
    "apply_fd004_phase_adjustment_630p",
    "apply_fd004_g1_right_ecdf_630p",
    "FD004_G1_REFERENCE_630P",
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "FD004_CLEAN_DEV_ADJUSTED_UNCERTAINTY_630P",
    "FD004_G1_EVIDENCE_CONTRACT_SHA256_630P",

    # -------------------------------------------------------------------------
    # G2 input weights
    # -------------------------------------------------------------------------
    "FD004_G2_RAW_EQUAL_ENGINE_WEIGHTS_630Q",
    "FD004_G2_INPUT_CONTRACT_SHA256_630Q",

    # -------------------------------------------------------------------------
    # G2 final reference / helper / clean risk
    # -------------------------------------------------------------------------
    "apply_fd004_g2_weighted_midrank_630r",
    "FD004_G2_REFERENCE_VALUES_630R",
    "FD004_G2_REFERENCE_WEIGHTS_630R",
    "FD004_G2_UNIQUE_VALUES_630R",
    "FD004_G2_UNIQUE_WEIGHTS_630R",
    "FD004_G2_CUMULATIVE_WEIGHTS_630R",
    "FD004_G2_MID_MASS_630R",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
    "FD004_SELECTIVE_RISK_CONTRACT_SHA256_630R",
)


rows_630v3 = []


for name_630v3 in candidate_names_630v3:

    present_630v3 = (
        name_630v3
        in globals()
    )

    if not present_630v3:

        rows_630v3.append(
            {
                "name":
                    name_630v3,

                "present":
                    False,

                "type":
                    None,

                "shape_or_len":
                    None,

                "dtype_or_signature":
                    None,
            }
        )

        continue


    obj_630v3 = globals()[
        name_630v3
    ]

    type_name_630v3 = type(
        obj_630v3
    ).__name__

    shape_or_len_630v3 = None
    dtype_or_signature_630v3 = None


    if isinstance(
        obj_630v3,
        np.ndarray,
    ):

        shape_or_len_630v3 = tuple(
            obj_630v3.shape
        )

        dtype_or_signature_630v3 = str(
            obj_630v3.dtype
        )


    elif isinstance(
        obj_630v3,
        pd.DataFrame,
    ):

        shape_or_len_630v3 = tuple(
            obj_630v3.shape
        )

        dtype_or_signature_630v3 = "DataFrame"


    elif isinstance(
        obj_630v3,
        (
            tuple,
            list,
        ),
    ):

        shape_or_len_630v3 = len(
            obj_630v3
        )

        dtype_or_signature_630v3 = (
            type_name_630v3
        )


    elif callable(
        obj_630v3
    ):

        try:

            dtype_or_signature_630v3 = str(
                inspect.signature(
                    obj_630v3
                )
            )

        except Exception:

            dtype_or_signature_630v3 = (
                "<signature unavailable>"
            )


    elif isinstance(
        obj_630v3,
        str,
    ):

        shape_or_len_630v3 = len(
            obj_630v3
        )

        dtype_or_signature_630v3 = (
            obj_630v3
        )


    elif isinstance(
        obj_630v3,
        (
            int,
            float,
            bool,
            np.integer,
            np.floating,
        ),
    ):

        dtype_or_signature_630v3 = repr(
            obj_630v3
        )


    else:

        dtype_or_signature_630v3 = (
            type_name_630v3
        )


    rows_630v3.append(
        {
            "name":
                name_630v3,

            "present":
                True,

            "type":
                type_name_630v3,

            "shape_or_len":
                shape_or_len_630v3,

            "dtype_or_signature":
                dtype_or_signature_630v3,
        }
    )


locator_df_630v3 = pd.DataFrame(
    rows_630v3
)


# =============================================================================
# 2) NAMES-ONLY FALLBACK
#
# Snapshot globals first — do not iterate over a mutating globals() view.
# =============================================================================

globals_snapshot_630v3 = tuple(
    globals().keys()
)


relevant_tokens_630v3 = (
    "MC",
    "PHASE",
    "G1",
    "G2",
    "RISK",
    "REFERENCE",
    "MID",
    "ECDF",
)


relevant_steps_630v3 = (
    "630L",
    "630M",
    "630N",
    "630O",
    "630P",
    "630Q",
    "630R",
)


fallback_names_630v3 = sorted(
    name_630v3
    for name_630v3
    in globals_snapshot_630v3

    if
    "FD004"
    in name_630v3.upper()

    and
    any(
        step_630v3
        in name_630v3.upper()
        for step_630v3
        in relevant_steps_630v3
    )

    and
    any(
        token_630v3
        in name_630v3.upper()
        for token_630v3
        in relevant_tokens_630v3
    )
)


# Function-name-only fallback.
function_fallback_630v3 = []


for name_630v3 in globals_snapshot_630v3:

    if not any(
        token_630v3
        in name_630v3.lower()
        for token_630v3
        in (
            "mc_dropout",
            "phase_adjust",
            "g1",
            "g2",
            "midrank",
            "ecdf",
        )
    ):
        continue


    obj_630v3 = globals().get(
        name_630v3
    )

    if callable(
        obj_630v3
    ):

        try:

            sig_630v3 = str(
                inspect.signature(
                    obj_630v3
                )
            )

        except Exception:

            sig_630v3 = (
                "<signature unavailable>"
            )


        function_fallback_630v3.append(
            (
                name_630v3,
                sig_630v3,
            )
        )


function_fallback_630v3 = sorted(
    function_fallback_630v3
)


# =============================================================================
# 3) STATE
# =============================================================================

FD004_FULL_RELIABILITY_CHAIN_LOCATOR_COMPLETED_630V3 = True

FD004_CORRUPTION_EXECUTED_630V3 = False
FD004_MODEL_INFERENCE_PERFORMED_630V3 = False
FD004_MC_EXECUTION_PERFORMED_630V3 = False
FD004_DEV_OUTCOMES_USED_630V3 = False
FD004_THRESHOLD_SELECTED_630V3 = False
FD004_POLICY_SELECTED_630V3 = False
FD004_TEST_CONTENTS_READ_630V3 = False
FD004_RUL_CONTENTS_READ_630V3 = False

all_checks_passed_630v3 = True


# =============================================================================
# 4) REPORT
# =============================================================================

print(
    "=" * 174
)

print(
    "STEP 6.30V-3 — FD004 FULL CORRUPTION "
    "RELIABILITY-CHAIN AUTHORITY LOCATOR"
)

print(
    "=" * 174
)


print(
    "Corruption executed              :",
    False,
)

print(
    "Model inference performed        :",
    False,
)

print(
    "MC execution performed           :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- EXPLICITLY WHITELISTED "
    "RELIABILITY-CHAIN AUTHORITIES ---"
)

print(
    locator_df_630v3.to_string(
        index=False
    )
)


print(
    "\n--- NAMES-ONLY FALLBACK: "
    "FD004 630L..630R RELIABILITY OBJECTS ---"
)

for name_630v3 in fallback_names_630v3:

    print(
        name_630v3
    )


print(
    "\n--- CALLABLE-ONLY FALLBACK ---"
)

for (
    name_630v3,
    sig_630v3,
) in function_fallback_630v3:

    print(
        f"{name_630v3}{sig_630v3}"
    )


print(
    "\nPASS — only metadata and names of already-resident "
    "FD004 reliability-chain authorities were inspected. "
    "No corruption, inference, MC, DEV scoring, threshold, "
    "TEST, or RUL operation occurred."
)

In [ ]:
# STEP 6.30V-4
# FD004 LABEL-FREE MC + PHASE→G1→G2 RELIABILITY-ENGINE FREEZE
# AND CLEAN-EVIDENCE EXACT-REPRODUCTION AUDIT
#
# NO corruption execution.
# NO new deterministic inference.
# NO new MC execution.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import hashlib
import json
import torch


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v3", False)):
    raise RuntimeError(
        "STEP 6.30V-3 must pass before STEP 6.30V-4."
    )

if not bool(globals().get("all_checks_passed_630v2", False)):
    raise RuntimeError(
        "STEP 6.30V-2 smoke bridge must remain passed."
    )


required_names_630v4 = (
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_STD_DDOF_630L",
    "FD004_PHASE_EDGES_630O",
    "apply_fd004_phase_adjustment_630p",
    "FD004_G1_REFERENCE_630P",
    "FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P",
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "apply_fd004_g2_weighted_midrank_630r",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_MC_STD_630M",
    "FD004_G2_UNIQUE_VALUES_630R",
    "FD004_G2_CUMULATIVE_WEIGHTS_630R",
    "FD004_G2_MID_MASS_630R",
)


missing_630v4 = [
    name_630v4
    for name_630v4 in required_names_630v4
    if name_630v4 not in globals()
]


if missing_630v4:
    raise RuntimeError(
        "Missing frozen reliability authority object(s): "
        + ", ".join(missing_630v4)
    )


if int(FD004_MC_PASSES_630L) != 50:
    raise RuntimeError(
        "Frozen MC pass count is not 50."
    )

if int(FD004_MC_BATCH_SIZE_630L) != 512:
    raise RuntimeError(
        "Frozen MC batch size is not 512."
    )

if int(FD004_MC_SEED_630L) != 42:
    raise RuntimeError(
        "Frozen MC seed is not 42."
    )

if int(FD004_MC_STD_DDOF_630L) != 0:
    raise RuntimeError(
        "Frozen MC STD ddof is not 0."
    )


# =============================================================================
# 1) CANONICAL HASH HELPER
# =============================================================================

def _sha_arr_630v4(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    h = hashlib.sha256()

    h.update(
        str(arr.dtype).encode("utf-8")
    )

    h.update(
        repr(tuple(arr.shape)).encode("utf-8")
    )

    h.update(
        arr.tobytes(order="C")
    )

    return h.hexdigest()


# =============================================================================
# 2) FREEZE EXPLICIT G1 RIGHT-ECDF MAPPING
# =============================================================================

FD004_G1_REFERENCE_AUTHORITY_630V4 = np.asarray(
    FD004_G1_REFERENCE_630P,
    dtype=np.float64,
).reshape(-1)


if FD004_G1_REFERENCE_AUTHORITY_630V4.shape != (10824,):
    raise RuntimeError(
        "Unexpected FD004 G1 reference shape: "
        f"{FD004_G1_REFERENCE_AUTHORITY_630V4.shape}"
    )


if not np.all(
    np.isfinite(
        FD004_G1_REFERENCE_AUTHORITY_630V4
    )
):
    raise RuntimeError(
        "FD004 G1 reference contains non-finite values."
    )


if np.any(
    np.diff(
        FD004_G1_REFERENCE_AUTHORITY_630V4
    ) < 0
):
    raise RuntimeError(
        "FD004 G1 reference is not sorted ascending."
    )


def apply_fd004_g1_right_ecdf_630v4(values):

    x_630v4 = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)


    if not np.all(
        np.isfinite(x_630v4)
    ):
        raise ValueError(
            "G1 input contains non-finite values."
        )


    right_630v4 = np.searchsorted(
        FD004_G1_REFERENCE_AUTHORITY_630V4,
        x_630v4,
        side="right",
    )


    result_630v4 = (
        right_630v4.astype(np.float64)
        / float(
            FD004_G1_REFERENCE_AUTHORITY_630V4.size
        )
    )


    return result_630v4


# =============================================================================
# 3) FREEZE CANONICAL MC-DROPOUT EXECUTION HELPER
#
# IMPORTANT:
# Defined here but NOT executed in STEP 6.30V-4.
# =============================================================================

def fd004_mc_dropout_predict_630v4(
    X,
    model,
    device,
):

    x_630v4 = np.asarray(
        X,
        dtype=np.float32,
    )


    if x_630v4.ndim != 3:
        raise ValueError(
            "MC input must be a 3-D tensor: "
            "(windows, cycles, features)."
        )


    passes_630v4 = int(
        FD004_MC_PASSES_630L
    )

    batch_630v4 = int(
        FD004_MC_BATCH_SIZE_630L
    )

    seed_630v4 = int(
        FD004_MC_SEED_630L
    )

    ddof_630v4 = int(
        FD004_MC_STD_DDOF_630L
    )


    n_630v4 = int(
        x_630v4.shape[0]
    )


    matrix_630v4 = np.empty(
        (
            passes_630v4,
            n_630v4,
        ),
        dtype=np.float32,
    )


    prior_training_mode_630v4 = bool(
        model.training
    )


    torch.manual_seed(
        seed_630v4
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed_630v4
        )


    model.train()


    try:

        with torch.no_grad():

            for pass_idx_630v4 in range(
                passes_630v4
            ):

                for start_630v4 in range(
                    0,
                    n_630v4,
                    batch_630v4,
                ):

                    stop_630v4 = min(
                        start_630v4
                        + batch_630v4,
                        n_630v4,
                    )


                    xb_630v4 = torch.from_numpy(
                        x_630v4[
                            start_630v4:
                            stop_630v4
                        ]
                    ).to(device)


                    pred_630v4 = (
                        model(
                            xb_630v4
                        )
                        .detach()
                        .cpu()
                        .numpy()
                        .reshape(-1)
                        .astype(
                            np.float32,
                            copy=False,
                        )
                    )


                    if pred_630v4.shape != (
                        stop_630v4
                        - start_630v4,
                    ):
                        raise RuntimeError(
                            "Unexpected MC prediction "
                            "batch shape."
                        )


                    matrix_630v4[
                        pass_idx_630v4,
                        start_630v4:
                        stop_630v4,
                    ] = pred_630v4


    finally:

        if prior_training_mode_630v4:
            model.train()

        else:
            model.eval()


    mean_630v4 = np.mean(
        matrix_630v4,
        axis=0,
        dtype=np.float64,
    )


    std_630v4 = np.std(
        matrix_630v4,
        axis=0,
        ddof=ddof_630v4,
        dtype=np.float64,
    )


    return (
        matrix_630v4,
        mean_630v4,
        std_630v4,
    )


# =============================================================================
# 4) FREEZE PHASE -> G1 -> G2 RELIABILITY HELPER
# =============================================================================

def apply_fd004_final_reliability_chain_630v4(
    deterministic_prediction,
    mc_std,
):

    pred_630v4 = np.asarray(
        deterministic_prediction,
        dtype=np.float64,
    ).reshape(-1)


    std_630v4 = np.asarray(
        mc_std,
        dtype=np.float64,
    ).reshape(-1)


    if pred_630v4.shape != std_630v4.shape:
        raise ValueError(
            "Deterministic prediction and MC STD "
            "must have identical shape."
        )


    if not (
        np.all(
            np.isfinite(pred_630v4)
        )
        and
        np.all(
            np.isfinite(std_630v4)
        )
    ):
        raise ValueError(
            "Reliability-chain inputs contain "
            "non-finite values."
        )


    adjusted_630v4 = np.asarray(
        apply_fd004_phase_adjustment_630p(
            pred_630v4,
            std_630v4,
        ),
        dtype=np.float64,
    ).reshape(-1)


    g1_630v4 = np.asarray(
        apply_fd004_g1_right_ecdf_630v4(
            adjusted_630v4
        ),
        dtype=np.float64,
    ).reshape(-1)


    final_risk_630v4 = np.asarray(
        apply_fd004_g2_weighted_midrank_630r(
            g1_630v4
        ),
        dtype=np.float64,
    ).reshape(-1)


    if not (
        adjusted_630v4.shape
        == g1_630v4.shape
        == final_risk_630v4.shape
        == pred_630v4.shape
    ):
        raise RuntimeError(
            "Reliability-chain output shape mismatch."
        )


    if not (
        np.all(
            np.isfinite(adjusted_630v4)
        )
        and
        np.all(
            np.isfinite(g1_630v4)
        )
        and
        np.all(
            np.isfinite(final_risk_630v4)
        )
    ):
        raise RuntimeError(
            "Reliability-chain output contains "
            "non-finite values."
        )


    if np.any(
        (g1_630v4 < 0.0)
        |
        (g1_630v4 > 1.0)
    ):
        raise RuntimeError(
            "G1 risk escaped [0,1]."
        )


    if np.any(
        (final_risk_630v4 < 0.0)
        |
        (final_risk_630v4 > 1.0)
    ):
        raise RuntimeError(
            "Final selective risk escaped [0,1]."
        )


    return (
        adjusted_630v4,
        g1_630v4,
        final_risk_630v4,
    )


# =============================================================================
# 5) CLEAN-EVIDENCE EXACT-REPRODUCTION AUDIT
#
# Uses existing frozen label-free clean-DEV prediction + MC STD only.
#
# NO inference.
# NO MC execution.
# =============================================================================

clean_pred_630v4 = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float64,
).reshape(-1)


clean_std_630v4 = np.asarray(
    FD004_CLEAN_DEV_MC_STD_630M,
    dtype=np.float64,
).reshape(-1)


(
    clean_adjusted_rebuilt_630v4,
    clean_g1_rebuilt_630v4,
    clean_final_rebuilt_630v4,
) = apply_fd004_final_reliability_chain_630v4(
    clean_pred_630v4,
    clean_std_630v4,
)


clean_adjusted_frozen_630v4 = np.asarray(
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P,
    dtype=np.float64,
).reshape(-1)


clean_g1_frozen_630v4 = np.asarray(
    FD004_CLEAN_DEV_G1_RISK_630P,
    dtype=np.float64,
).reshape(-1)


clean_final_frozen_630v4 = np.asarray(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
    dtype=np.float64,
).reshape(-1)


adjusted_exact_630v4 = np.array_equal(
    clean_adjusted_rebuilt_630v4,
    clean_adjusted_frozen_630v4,
)


g1_exact_630v4 = np.array_equal(
    clean_g1_rebuilt_630v4,
    clean_g1_frozen_630v4,
)


final_exact_630v4 = np.array_equal(
    clean_final_rebuilt_630v4,
    clean_final_frozen_630v4,
)


if not adjusted_exact_630v4:
    raise RuntimeError(
        "Rebuilt clean phase-adjusted uncertainty "
        "is not exact."
    )


if not g1_exact_630v4:
    raise RuntimeError(
        "Rebuilt clean G1 risk is not exact."
    )


if not final_exact_630v4:
    raise RuntimeError(
        "Rebuilt clean final selective risk "
        "is not exact."
    )


# =============================================================================
# 6) FREEZE ENGINE CONTRACT
# =============================================================================

FD004_RELIABILITY_ENGINE_CONTRACT_630V4 = {

    "dataset":
        "FD004",

    "stage":
        "6.30V-4",

    "role":
        "label_free_corruption_reliability_engine",

    "mc_passes":
        int(FD004_MC_PASSES_630L),

    "mc_batch_size":
        int(FD004_MC_BATCH_SIZE_630L),

    "mc_seed":
        int(FD004_MC_SEED_630L),

    "mc_std_ddof":
        int(FD004_MC_STD_DDOF_630L),

    "mc_matrix_dtype":
        "float32",

    "mc_mean_dtype":
        "float64",

    "mc_std_dtype":
        "float64",

    "phase_adjustment_helper":
        "apply_fd004_phase_adjustment_630p",

    "g1_mapping":
        "right_empirical_cdf",

    "g1_reference":
        "FD004_G1_REFERENCE_630P",

    "g2_mapping_helper":
        "apply_fd004_g2_weighted_midrank_630r",

    "final_risk_orientation":
        "lower_is_more_reliable",

    "dev_labels_or_errors_used":
        False,

    "threshold_selected":
        False,

    "test_used":
        False,

    "official_rul_used":
        False,
}


FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4 = (
    hashlib.sha256(
        json.dumps(
            FD004_RELIABILITY_ENGINE_CONTRACT_630V4,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


FD004_G1_REFERENCE_AUTHORITY_SHA256_630V4 = (
    _sha_arr_630v4(
        FD004_G1_REFERENCE_AUTHORITY_630V4
    )
)


FD004_CLEAN_REBUILT_ADJUSTED_SHA256_630V4 = (
    _sha_arr_630v4(
        clean_adjusted_rebuilt_630v4
    )
)


FD004_CLEAN_REBUILT_G1_SHA256_630V4 = (
    _sha_arr_630v4(
        clean_g1_rebuilt_630v4
    )
)


FD004_CLEAN_REBUILT_FINAL_RISK_SHA256_630V4 = (
    _sha_arr_630v4(
        clean_final_rebuilt_630v4
    )
)


FD004_RELIABILITY_ENGINE_FROZEN_630V4 = True

FD004_CLEAN_RELIABILITY_REPRODUCTION_EXACT_630V4 = True


FD004_CORRUPTION_EXECUTED_630V4 = False

FD004_NEW_MODEL_INFERENCE_PERFORMED_630V4 = False

FD004_NEW_MC_EXECUTION_PERFORMED_630V4 = False

FD004_DEV_TARGETS_ERRORS_USED_630V4 = False

FD004_THRESHOLD_SELECTED_630V4 = False

FD004_POLICY_SELECTED_630V4 = False

FD004_TEST_CONTENTS_READ_630V4 = False

FD004_RUL_CONTENTS_READ_630V4 = False


all_checks_passed_630v4 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print(
    "=" * 176
)

print(
    "STEP 6.30V-4 — FD004 LABEL-FREE MC + PHASE→G1→G2 "
    "RELIABILITY-ENGINE FREEZE + CLEAN EXACT-REPRODUCTION AUDIT"
)

print(
    "=" * 176
)


print(
    "Corruption executed              :",
    False,
)

print(
    "New model inference performed    :",
    False,
)

print(
    "New MC execution performed       :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- FROZEN MC EXECUTION SEMANTICS ---"
)

print(
    "passes / batch / seed / ddof     :",
    (
        int(FD004_MC_PASSES_630L),
        int(FD004_MC_BATCH_SIZE_630L),
        int(FD004_MC_SEED_630L),
        int(FD004_MC_STD_DDOF_630L),
    ),
)

print(
    "MC matrix dtype                  : float32"
)

print(
    "MC mean / STD dtype              : float64 / float64"
)

print(
    "MC helper executed in this step  :",
    False,
)


print(
    "\n--- FROZEN RELIABILITY CHAIN ---"
)

print(
    "Phase helper                     : "
    "apply_fd004_phase_adjustment_630p"
)

print(
    "G1 mapping                       : "
    "right empirical CDF"
)

print(
    "G1 reference size                :",
    FD004_G1_REFERENCE_AUTHORITY_630V4.size,
)

print(
    "G2 helper                        : "
    "apply_fd004_g2_weighted_midrank_630r"
)

print(
    "Risk orientation                 : "
    "lower = more reliable"
)


print(
    "\n--- CLEAN EXACT-REPRODUCTION AUDIT ---"
)

print(
    "Clean population                 :",
    clean_pred_630v4.size,
)

print(
    "Phase-adjusted uncertainty exact :",
    adjusted_exact_630v4,
)

print(
    "G1 risk exact                    :",
    g1_exact_630v4,
)

print(
    "Final G2 selective risk exact    :",
    final_exact_630v4,
)

print(
    "Final risk min/median/mean/max   :",
    (
        float(
            np.min(
                clean_final_rebuilt_630v4
            )
        ),
        float(
            np.median(
                clean_final_rebuilt_630v4
            )
        ),
        float(
            np.mean(
                clean_final_rebuilt_630v4
            )
        ),
        float(
            np.max(
                clean_final_rebuilt_630v4
            )
        ),
    ),
)


print(
    "\n--- EVIDENCE FINGERPRINTS ---"
)

print(
    "G1 reference SHA-256             :",
    FD004_G1_REFERENCE_AUTHORITY_SHA256_630V4,
)

print(
    "Rebuilt adjusted SHA-256         :",
    FD004_CLEAN_REBUILT_ADJUSTED_SHA256_630V4,
)

print(
    "Rebuilt G1 SHA-256               :",
    FD004_CLEAN_REBUILT_G1_SHA256_630V4,
)

print(
    "Rebuilt final risk SHA-256       :",
    FD004_CLEAN_REBUILT_FINAL_RISK_SHA256_630V4,
)

print(
    "Reliability-engine contract SHA-256:",
    FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4,
)


print(
    "\nPASS — the FD004 label-free MC protocol and "
    "phase→G1→G2 reliability engine are now explicitly frozen. "
    "The already-frozen clean-DEV reliability evidence was "
    "reproduced exactly without running inference or MC. "
    "No DEV outcomes, threshold, TEST, or RUL information was used."
)


print(
    "Next: bind this frozen reliability engine to the "
    "already-validated corruption→preprocessing→window→model "
    "bridge and freeze the complete 381-scenario execution "
    "contract before any full corruption run."
)

In [ ]:
# STEP 6.30V-4R1
# FD004 CLEAN RELIABILITY-HELPER RETURN-SEMANTICS AUDIT
# No corruption, inference, MC, DEV labels/errors, threshold, TEST, or RUL.

import numpy as np

required_630v4r1 = (
    "apply_fd004_phase_adjustment_630p",
    "apply_fd004_g2_weighted_midrank_630r",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_MC_STD_630M",
    "FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P",
    "FD004_G1_REFERENCE_630P",
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
)

missing_630v4r1 = [
    name for name in required_630v4r1
    if name not in globals()
]

if missing_630v4r1:
    raise RuntimeError(
        "Missing STEP 6.30V-4R1 authority object(s): "
        + ", ".join(missing_630v4r1)
    )


clean_pred_630v4r1 = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K
).reshape(-1)

clean_std_630v4r1 = np.asarray(
    FD004_CLEAN_DEV_MC_STD_630M
).reshape(-1)

frozen_adjusted_630v4r1 = np.asarray(
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P
).reshape(-1)

frozen_g1_630v4r1 = np.asarray(
    FD004_CLEAN_DEV_G1_RISK_630P
).reshape(-1)

frozen_final_630v4r1 = np.asarray(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R
).reshape(-1)


# Existing frozen helpers only — no model/MC execution.
phase_raw_630v4r1 = apply_fd004_phase_adjustment_630p(
    clean_pred_630v4r1,
    clean_std_630v4r1,
)

g2_raw_630v4r1 = apply_fd004_g2_weighted_midrank_630r(
    frozen_g1_630v4r1
)


def describe_return_630v4r1(name, value, reference=None):

    print(f"\n{name}")
    print("  python type :", type(value).__name__)

    if isinstance(value, (tuple, list)):

        print("  item count  :", len(value))

        for idx, item in enumerate(value):

            arr = np.asarray(item)

            print(
                f"  item {idx}: "
                f"type={type(item).__name__}, "
                f"shape={arr.shape}, "
                f"dtype={arr.dtype}"
            )

            if (
                reference is not None
                and arr.shape == reference.shape
            ):
                print(
                    "           exact reference match=",
                    np.array_equal(
                        arr,
                        reference,
                    ),
                )

    else:

        arr = np.asarray(value)

        print("  shape       :", arr.shape)
        print("  dtype       :", arr.dtype)

        if (
            reference is not None
            and arr.shape == reference.shape
        ):
            print(
                "  exact reference match :",
                np.array_equal(
                    arr,
                    reference,
                ),
            )


describe_return_630v4r1(
    "Phase-adjustment helper return",
    phase_raw_630v4r1,
    frozen_adjusted_630v4r1,
)

describe_return_630v4r1(
    "G2 helper return on frozen clean G1",
    g2_raw_630v4r1,
    frozen_final_630v4r1,
)


# Independently verify the frozen G1 right-ECDF semantics.
g1_reference_630v4r1 = np.asarray(
    FD004_G1_REFERENCE_630P,
    dtype=np.float64,
).reshape(-1)


g1_right_candidate_630v4r1 = (
    np.searchsorted(
        g1_reference_630v4r1,
        frozen_adjusted_630v4r1,
        side="right",
    ).astype(np.float64)
    /
    float(
        g1_reference_630v4r1.size
    )
)


print("\nG1 right-ECDF semantic check")

print(
    "  candidate shape :",
    g1_right_candidate_630v4r1.shape,
)

print(
    "  exact frozen G1 match :",
    np.array_equal(
        g1_right_candidate_630v4r1,
        frozen_g1_630v4r1,
    ),
)

print(
    "  max abs diff :",
    float(
        np.max(
            np.abs(
                g1_right_candidate_630v4r1
                -
                frozen_g1_630v4r1
            )
        )
    ),
)


FD004_RELIABILITY_HELPER_RETURN_AUDIT_COMPLETED_630V4R1 = True

all_checks_passed_630v4r1 = True


print("\n" + "=" * 170)

print(
    "STEP 6.30V-4R1 — FD004 CLEAN RELIABILITY-HELPER "
    "RETURN-SEMANTICS AUDIT"
)

print("=" * 170)

print("Corruption executed           : False")
print("New model inference performed : False")
print("New MC execution performed    : False")
print("DEV targets/errors used       : False")
print("Threshold selected            : False")
print("Policy selected               : False")
print("TEST contents read            : False")
print("Official RUL contents read    : False")

print(
    "\nPASS — only already-frozen clean label-free evidence "
    "was passed through already-frozen reliability helpers "
    "to identify their exact return structure. "
    "No experimental decision was made."
)

In [ ]:
# STEP 6.30V-4R2
# FD004 LABEL-FREE RELIABILITY-ENGINE CORRECTION + EXACT CLEAN REPRODUCTION LOCK

import numpy as np
import hashlib
import json
import torch


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v4r1", False)):
    raise RuntimeError(
        "STEP 6.30V-4R1 must pass before STEP 6.30V-4R2."
    )


required_630v4r2 = (
    "FD004_MC_PASSES_630L",
    "FD004_MC_BATCH_SIZE_630L",
    "FD004_MC_SEED_630L",
    "FD004_MC_STD_DDOF_630L",
    "apply_fd004_phase_adjustment_630p",
    "FD004_G1_REFERENCE_630P",
    "apply_fd004_g2_weighted_midrank_630r",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "FD004_CLEAN_DEV_MC_STD_630M",
    "FD004_CLEAN_DEV_PHASE_IDS_630P",
    "FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P",
    "FD004_CLEAN_DEV_G1_RISK_630P",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
)


missing_630v4r2 = [
    name
    for name in required_630v4r2
    if name not in globals()
]


if missing_630v4r2:
    raise RuntimeError(
        "Missing STEP 6.30V-4R2 authority object(s): "
        + ", ".join(missing_630v4r2)
    )


if (
    int(FD004_MC_PASSES_630L) != 50
    or int(FD004_MC_BATCH_SIZE_630L) != 512
    or int(FD004_MC_SEED_630L) != 42
    or int(FD004_MC_STD_DDOF_630L) != 0
):
    raise RuntimeError(
        "Frozen MC protocol values differ from the locked authority."
    )


# =============================================================================
# 1) HASH HELPER
# =============================================================================

def _sha_arr_630v4r2(value):

    arr = np.ascontiguousarray(
        np.asarray(value)
    )

    h = hashlib.sha256()

    h.update(
        str(arr.dtype).encode("utf-8")
    )

    h.update(
        repr(tuple(arr.shape)).encode("utf-8")
    )

    h.update(
        arr.tobytes(order="C")
    )

    return h.hexdigest()


# =============================================================================
# 2) EXACT G1 RIGHT-ECDF AUTHORITY
# =============================================================================

FD004_G1_REFERENCE_AUTHORITY_630V4R2 = np.asarray(
    FD004_G1_REFERENCE_630P,
    dtype=np.float64,
).reshape(-1)


if FD004_G1_REFERENCE_AUTHORITY_630V4R2.shape != (10824,):
    raise RuntimeError(
        "Unexpected frozen G1-reference shape."
    )


if not np.all(
    np.isfinite(
        FD004_G1_REFERENCE_AUTHORITY_630V4R2
    )
):
    raise RuntimeError(
        "Frozen G1 reference contains non-finite values."
    )


if np.any(
    np.diff(
        FD004_G1_REFERENCE_AUTHORITY_630V4R2
    ) < 0
):
    raise RuntimeError(
        "Frozen G1 reference is not sorted ascending."
    )


def apply_fd004_g1_right_ecdf_630v4r2(values):

    x = np.asarray(
        values,
        dtype=np.float64,
    ).reshape(-1)


    if not np.all(np.isfinite(x)):
        raise ValueError(
            "G1 input contains non-finite values."
        )


    right = np.searchsorted(
        FD004_G1_REFERENCE_AUTHORITY_630V4R2,
        x,
        side="right",
    )


    return (
        right.astype(np.float64)
        /
        float(
            FD004_G1_REFERENCE_AUTHORITY_630V4R2.size
        )
    )


# =============================================================================
# 3) CANONICAL MC HELPER
#
# Defined now, but NOT executed in this step.
# =============================================================================

def fd004_mc_dropout_predict_630v4r2(
    X,
    model,
    device,
):

    x = np.asarray(
        X,
        dtype=np.float32,
    )


    if x.ndim != 3:
        raise ValueError(
            "MC input must have shape "
            "(windows, cycles, features)."
        )


    passes = int(
        FD004_MC_PASSES_630L
    )

    batch_size = int(
        FD004_MC_BATCH_SIZE_630L
    )

    seed = int(
        FD004_MC_SEED_630L
    )

    ddof = int(
        FD004_MC_STD_DDOF_630L
    )


    n = int(
        x.shape[0]
    )


    matrix = np.empty(
        (
            passes,
            n,
        ),
        dtype=np.float32,
    )


    prior_mode = bool(
        model.training
    )


    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    model.train()


    try:

        with torch.no_grad():

            for pass_idx in range(
                passes
            ):

                for start in range(
                    0,
                    n,
                    batch_size,
                ):

                    stop = min(
                        start + batch_size,
                        n,
                    )


                    xb = torch.from_numpy(
                        x[
                            start:
                            stop
                        ]
                    ).to(
                        device
                    )


                    pred = (
                        model(
                            xb
                        )
                        .detach()
                        .cpu()
                        .numpy()
                        .reshape(-1)
                        .astype(
                            np.float32,
                            copy=False,
                        )
                    )


                    if pred.shape != (
                        stop - start,
                    ):
                        raise RuntimeError(
                            "Unexpected MC prediction batch shape."
                        )


                    matrix[
                        pass_idx,
                        start:
                        stop,
                    ] = pred


    finally:

        if prior_mode:
            model.train()

        else:
            model.eval()


    mean = np.mean(
        matrix,
        axis=0,
        dtype=np.float64,
    )


    std = np.std(
        matrix,
        axis=0,
        ddof=ddof,
        dtype=np.float64,
    )


    return (
        matrix,
        mean,
        std,
    )


# =============================================================================
# 4) CORRECTED PHASE -> G1 -> G2 RELIABILITY CHAIN
# =============================================================================

def apply_fd004_final_reliability_chain_630v4r2(
    deterministic_prediction,
    mc_std,
):

    pred = np.asarray(
        deterministic_prediction,
        dtype=np.float64,
    ).reshape(-1)


    std = np.asarray(
        mc_std,
        dtype=np.float64,
    ).reshape(-1)


    if pred.shape != std.shape:
        raise ValueError(
            "Prediction and MC-STD shapes differ."
        )


    if not (
        np.all(np.isfinite(pred))
        and
        np.all(np.isfinite(std))
    ):
        raise ValueError(
            "Reliability-chain input contains "
            "non-finite values."
        )


    phase_output = (
        apply_fd004_phase_adjustment_630p(
            pred,
            std,
        )
    )


    if (
        not isinstance(
            phase_output,
            tuple,
        )
        or
        len(
            phase_output
        ) != 2
    ):
        raise RuntimeError(
            "Frozen phase helper return structure changed; "
            "expected (phase_ids, adjusted_uncertainty)."
        )


    phase_ids = np.asarray(
        phase_output[0],
        dtype=np.int64,
    ).reshape(-1)


    adjusted = np.asarray(
        phase_output[1],
        dtype=np.float64,
    ).reshape(-1)


    if (
        phase_ids.shape != pred.shape
        or
        adjusted.shape != pred.shape
    ):
        raise RuntimeError(
            "Frozen phase-helper output shape mismatch."
        )


    g1 = np.asarray(
        apply_fd004_g1_right_ecdf_630v4r2(
            adjusted
        ),
        dtype=np.float64,
    ).reshape(-1)


    final_risk = np.asarray(
        apply_fd004_g2_weighted_midrank_630r(
            g1
        ),
        dtype=np.float64,
    ).reshape(-1)


    if not (
        phase_ids.shape
        ==
        adjusted.shape
        ==
        g1.shape
        ==
        final_risk.shape
        ==
        pred.shape
    ):
        raise RuntimeError(
            "Reliability-chain output shape mismatch."
        )


    if not (
        np.all(np.isfinite(adjusted))
        and
        np.all(np.isfinite(g1))
        and
        np.all(np.isfinite(final_risk))
    ):
        raise RuntimeError(
            "Reliability-chain output contains "
            "non-finite values."
        )


    if np.any(
        (g1 < 0.0)
        |
        (g1 > 1.0)
    ):
        raise RuntimeError(
            "G1 risk escaped [0,1]."
        )


    if np.any(
        (final_risk < 0.0)
        |
        (final_risk > 1.0)
    ):
        raise RuntimeError(
            "Final selective risk escaped [0,1]."
        )


    return (
        phase_ids,
        adjusted,
        g1,
        final_risk,
    )


# =============================================================================
# 5) EXACT CLEAN REPRODUCTION AUDIT
#
# Existing frozen clean evidence only.
# NO inference.
# NO MC.
# =============================================================================

clean_pred_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float64,
).reshape(-1)


clean_std_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_MC_STD_630M,
    dtype=np.float64,
).reshape(-1)


(
    clean_phase_ids_rebuilt_630v4r2,
    clean_adjusted_rebuilt_630v4r2,
    clean_g1_rebuilt_630v4r2,
    clean_final_rebuilt_630v4r2,
) = apply_fd004_final_reliability_chain_630v4r2(
    clean_pred_630v4r2,
    clean_std_630v4r2,
)


clean_phase_ids_frozen_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_PHASE_IDS_630P,
    dtype=np.int64,
).reshape(-1)


clean_adjusted_frozen_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_PHASE_ADJUSTED_UNCERTAINTY_630P,
    dtype=np.float64,
).reshape(-1)


clean_g1_frozen_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_G1_RISK_630P,
    dtype=np.float64,
).reshape(-1)


clean_final_frozen_630v4r2 = np.asarray(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
    dtype=np.float64,
).reshape(-1)


phase_ids_exact_630v4r2 = np.array_equal(
    clean_phase_ids_rebuilt_630v4r2,
    clean_phase_ids_frozen_630v4r2,
)


adjusted_exact_630v4r2 = np.array_equal(
    clean_adjusted_rebuilt_630v4r2,
    clean_adjusted_frozen_630v4r2,
)


g1_exact_630v4r2 = np.array_equal(
    clean_g1_rebuilt_630v4r2,
    clean_g1_frozen_630v4r2,
)


final_exact_630v4r2 = np.array_equal(
    clean_final_rebuilt_630v4r2,
    clean_final_frozen_630v4r2,
)


if not phase_ids_exact_630v4r2:
    raise RuntimeError(
        "Rebuilt clean phase IDs are not exact."
    )


if not adjusted_exact_630v4r2:
    raise RuntimeError(
        "Rebuilt clean phase-adjusted uncertainty is not exact."
    )


if not g1_exact_630v4r2:
    raise RuntimeError(
        "Rebuilt clean G1 risk is not exact."
    )


if not final_exact_630v4r2:
    raise RuntimeError(
        "Rebuilt clean final selective risk is not exact."
    )


# =============================================================================
# 6) FREEZE ENGINE CONTRACT
# =============================================================================

FD004_RELIABILITY_ENGINE_CONTRACT_630V4R2 = {

    "dataset":
        "FD004",

    "stage":
        "6.30V-4R2",

    "role":
        "label_free_corruption_reliability_engine",

    "mc_passes":
        int(
            FD004_MC_PASSES_630L
        ),

    "mc_batch_size":
        int(
            FD004_MC_BATCH_SIZE_630L
        ),

    "mc_seed":
        int(
            FD004_MC_SEED_630L
        ),

    "mc_std_ddof":
        int(
            FD004_MC_STD_DDOF_630L
        ),

    "mc_matrix_dtype":
        "float32",

    "mc_mean_dtype":
        "float64",

    "mc_std_dtype":
        "float64",

    "phase_helper":
        "apply_fd004_phase_adjustment_630p",

    "phase_helper_return":
        "(phase_ids, adjusted_uncertainty)",

    "g1_mapping":
        "right_empirical_cdf",

    "g1_reference":
        "FD004_G1_REFERENCE_630P",

    "g2_helper":
        "apply_fd004_g2_weighted_midrank_630r",

    "risk_orientation":
        "lower_is_more_reliable",

    "clean_reproduction_exact":
        True,

    "dev_labels_or_errors_used":
        False,

    "threshold_selected":
        False,

    "test_used":
        False,

    "official_rul_used":
        False,
}


FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4R2 = (
    hashlib.sha256(
        json.dumps(
            FD004_RELIABILITY_ENGINE_CONTRACT_630V4R2,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


FD004_G1_REFERENCE_AUTHORITY_SHA256_630V4R2 = (
    _sha_arr_630v4r2(
        FD004_G1_REFERENCE_AUTHORITY_630V4R2
    )
)


FD004_CLEAN_REBUILT_PHASE_IDS_SHA256_630V4R2 = (
    _sha_arr_630v4r2(
        clean_phase_ids_rebuilt_630v4r2
    )
)


FD004_CLEAN_REBUILT_ADJUSTED_SHA256_630V4R2 = (
    _sha_arr_630v4r2(
        clean_adjusted_rebuilt_630v4r2
    )
)


FD004_CLEAN_REBUILT_G1_SHA256_630V4R2 = (
    _sha_arr_630v4r2(
        clean_g1_rebuilt_630v4r2
    )
)


FD004_CLEAN_REBUILT_FINAL_RISK_SHA256_630V4R2 = (
    _sha_arr_630v4r2(
        clean_final_rebuilt_630v4r2
    )
)


FD004_RELIABILITY_ENGINE_FROZEN_630V4R2 = True

FD004_CLEAN_RELIABILITY_REPRODUCTION_EXACT_630V4R2 = True


FD004_CORRUPTION_EXECUTED_630V4R2 = False
FD004_NEW_MODEL_INFERENCE_PERFORMED_630V4R2 = False
FD004_NEW_MC_EXECUTION_PERFORMED_630V4R2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V4R2 = False
FD004_THRESHOLD_SELECTED_630V4R2 = False
FD004_POLICY_SELECTED_630V4R2 = False
FD004_TEST_CONTENTS_READ_630V4R2 = False
FD004_RUL_CONTENTS_READ_630V4R2 = False


all_checks_passed_630v4r2 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-4R2 — FD004 LABEL-FREE RELIABILITY-ENGINE "
    "CORRECTION + EXACT CLEAN REPRODUCTION LOCK"
)

print("=" * 176)


print(
    "Corruption executed              :",
    False,
)

print(
    "New model inference performed    :",
    False,
)

print(
    "New MC execution performed       :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- RESOLVED PHASE-HELPER SEMANTICS ---"
)

print(
    "Frozen return structure          : "
    "(phase_ids, adjusted_uncertainty)"
)

print(
    "Phase-ID dtype                   :",
    clean_phase_ids_rebuilt_630v4r2.dtype,
)

print(
    "Adjusted-uncertainty dtype       :",
    clean_adjusted_rebuilt_630v4r2.dtype,
)


print(
    "\n--- FROZEN MC EXECUTION SEMANTICS ---"
)

print(
    "passes / batch / seed / ddof     :",
    (
        int(
            FD004_MC_PASSES_630L
        ),
        int(
            FD004_MC_BATCH_SIZE_630L
        ),
        int(
            FD004_MC_SEED_630L
        ),
        int(
            FD004_MC_STD_DDOF_630L
        ),
    ),
)

print(
    "MC helper executed in this step  :",
    False,
)


print(
    "\n--- CLEAN EXACT-REPRODUCTION AUDIT ---"
)

print(
    "Clean population                 :",
    clean_pred_630v4r2.size,
)

print(
    "Phase IDs exact                  :",
    phase_ids_exact_630v4r2,
)

print(
    "Adjusted uncertainty exact       :",
    adjusted_exact_630v4r2,
)

print(
    "G1 right-ECDF exact              :",
    g1_exact_630v4r2,
)

print(
    "Final G2 selective risk exact    :",
    final_exact_630v4r2,
)

print(
    "Final risk min/median/mean/max   :",
    (
        float(
            np.min(
                clean_final_rebuilt_630v4r2
            )
        ),
        float(
            np.median(
                clean_final_rebuilt_630v4r2
            )
        ),
        float(
            np.mean(
                clean_final_rebuilt_630v4r2
            )
        ),
        float(
            np.max(
                clean_final_rebuilt_630v4r2
            )
        ),
    ),
)


print(
    "\n--- EVIDENCE FINGERPRINTS ---"
)

print(
    "G1 reference SHA-256             :",
    FD004_G1_REFERENCE_AUTHORITY_SHA256_630V4R2,
)

print(
    "Rebuilt phase IDs SHA-256        :",
    FD004_CLEAN_REBUILT_PHASE_IDS_SHA256_630V4R2,
)

print(
    "Rebuilt adjusted SHA-256         :",
    FD004_CLEAN_REBUILT_ADJUSTED_SHA256_630V4R2,
)

print(
    "Rebuilt G1 SHA-256               :",
    FD004_CLEAN_REBUILT_G1_SHA256_630V4R2,
)

print(
    "Rebuilt final risk SHA-256       :",
    FD004_CLEAN_REBUILT_FINAL_RISK_SHA256_630V4R2,
)

print(
    "Reliability-engine contract SHA-256:",
    FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4R2,
)


print(
    "\nPASS — the phase-helper return semantics are now "
    "resolved without ambiguity, and the complete frozen "
    "phase→G1→G2 clean reliability chain reproduces the "
    "established FD004 clean evidence exactly. "
    "No inference, MC, DEV outcomes, threshold, TEST, "
    "or RUL operation occurred."
)

print(
    "Next: bind this verified reliability engine to the "
    "already-validated corruption→preprocessing→window→model "
    "bridge and freeze the complete 381-scenario execution "
    "contract before running the full corruption experiment."
)

In [ ]:
# STEP 6.30V-5
# FD004 FULL-EXECUTION BRIDGE + MANIFEST AUTHORITY LOCATOR
#
# Metadata / names only.
# NO corruption.
# NO model inference.
# NO MC.
# NO DEV labels/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import pandas as pd
import inspect


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v4r2", False)):
    raise RuntimeError(
        "STEP 6.30V-4R2 must pass before STEP 6.30V-5."
    )

if not bool(globals().get("all_checks_passed_630v2", False)):
    raise RuntimeError(
        "STEP 6.30V-2 smoke bridge must remain passed."
    )

for flag_name_630v5 in (
    "FD004_DEV_TARGETS_ERRORS_USED_630V4R2",
    "FD004_THRESHOLD_SELECTED_630V4R2",
    "FD004_TEST_CONTENTS_READ_630V4R2",
    "FD004_RUL_CONTENTS_READ_630V4R2",
):
    if bool(globals().get(flag_name_630v5, True)):
        raise RuntimeError(
            f"Unexpected state before execution-authority locator: "
            f"{flag_name_630v5}=True."
        )


# =============================================================================
# 1) EXPLICIT WHITELIST OF LIKELY AUTHORITIES
# =============================================================================

candidate_names_630v5 = (

    # Manifest / admissibility authorities
    "FD004_CORRUPTION_MANIFEST_630S",
    "FD004_ANNOTATED_MANIFEST_630U",
    "FD004_ANNOTATED_CORRUPTION_MANIFEST_630U",
    "FD004_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_ADMISSIBLE_EXECUTION_MANIFEST_630U",
    "FD004_INADMISSIBLE_MANIFEST_630U",
    "FD004_EXECUTION_MANIFEST_SHA256_630U",
    "FD004_ADMISSIBILITY_CONTRACT_SHA256_630U",

    # FIT-only corruption references
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",
    "FD004_REGIME_SENSOR_STATS_630T",
    "FD004_REFERENCE_CONTRACT_SHA256_630T",

    # Frozen DEV / preprocessing / window authorities
    "FD004_DEV_RAW_630C",
    "FD004_DEV_NORMALIZED_630F",
    "X_DEV_FD004_630G",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",

    # Model authorities
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_TRAINED_MODEL_SHA256_630J",

    # Smoke bridge artifacts / contracts
    "FD004_SMOKE_SCENARIO_630V2",
    "FD004_SMOKE_SCENARIO_ROW_630V2",
    "FD004_SMOKE_X_630V2",
    "FD004_SMOKE_PREDICTION_630V2",
    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
    "FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2",
)


rows_630v5 = []


for name_630v5 in candidate_names_630v5:

    present_630v5 = (
        name_630v5 in globals()
    )

    if not present_630v5:

        rows_630v5.append(
            {
                "name":
                    name_630v5,

                "present":
                    False,

                "type":
                    None,

                "shape_or_len":
                    None,

                "dtype_or_signature":
                    None,
            }
        )

        continue


    obj_630v5 = globals()[
        name_630v5
    ]

    type_name_630v5 = type(
        obj_630v5
    ).__name__

    shape_or_len_630v5 = None

    dtype_or_signature_630v5 = None


    if isinstance(
        obj_630v5,
        np.ndarray,
    ):

        shape_or_len_630v5 = tuple(
            obj_630v5.shape
        )

        dtype_or_signature_630v5 = str(
            obj_630v5.dtype
        )


    elif isinstance(
        obj_630v5,
        pd.DataFrame,
    ):

        shape_or_len_630v5 = tuple(
            obj_630v5.shape
        )

        dtype_or_signature_630v5 = (
            "DataFrame"
        )


    elif isinstance(
        obj_630v5,
        pd.Series,
    ):

        shape_or_len_630v5 = tuple(
            obj_630v5.shape
        )

        dtype_or_signature_630v5 = str(
            obj_630v5.dtype
        )


    elif isinstance(
        obj_630v5,
        (
            tuple,
            list,
            dict,
        ),
    ):

        shape_or_len_630v5 = len(
            obj_630v5
        )

        dtype_or_signature_630v5 = (
            type_name_630v5
        )


    elif callable(
        obj_630v5
    ):

        try:

            dtype_or_signature_630v5 = str(
                inspect.signature(
                    obj_630v5
                )
            )

        except Exception:

            dtype_or_signature_630v5 = (
                "<signature unavailable>"
            )


    elif isinstance(
        obj_630v5,
        str,
    ):

        shape_or_len_630v5 = len(
            obj_630v5
        )

        dtype_or_signature_630v5 = (
            "str"
        )


    else:

        dtype_or_signature_630v5 = (
            type_name_630v5
        )


    rows_630v5.append(
        {
            "name":
                name_630v5,

            "present":
                True,

            "type":
                type_name_630v5,

            "shape_or_len":
                shape_or_len_630v5,

            "dtype_or_signature":
                dtype_or_signature_630v5,
        }
    )


locator_df_630v5 = pd.DataFrame(
    rows_630v5
)


# =============================================================================
# 2) NAMES-ONLY FALLBACK
# =============================================================================

globals_snapshot_630v5 = tuple(
    globals().keys()
)


relevant_step_tokens_630v5 = (
    "630T",
    "630U",
    "630V2",
)


relevant_name_tokens_630v5 = (
    "MANIFEST",
    "ADMISS",
    "CORRUPT",
    "REFERENCE",
    "SCALE",
    "IMPUT",
    "SMOKE",
    "BRIDGE",
    "NORMAL",
    "WINDOW",
    "PRED",
    "MODEL",
)


fallback_names_630v5 = sorted(
    name_630v5
    for name_630v5
    in globals_snapshot_630v5

    if
    "FD004"
    in name_630v5.upper()

    and
    any(
        step_630v5
        in name_630v5.upper()
        for step_630v5
        in relevant_step_tokens_630v5
    )

    and
    any(
        token_630v5
        in name_630v5.upper()
        for token_630v5
        in relevant_name_tokens_630v5
    )
)


# =============================================================================
# 3) CALLABLE-ONLY FALLBACK
# =============================================================================

callable_fallback_630v5 = []


for name_630v5 in globals_snapshot_630v5:

    lower_name_630v5 = (
        name_630v5.lower()
    )


    if not any(
        token_630v5
        in lower_name_630v5

        for token_630v5
        in (
            "corrupt",
            "normal",
            "window",
            "predict",
            "bridge",
            "imput",
        )
    ):

        continue


    obj_630v5 = globals().get(
        name_630v5
    )


    if callable(
        obj_630v5
    ):

        try:

            sig_630v5 = str(
                inspect.signature(
                    obj_630v5
                )
            )

        except Exception:

            sig_630v5 = (
                "<signature unavailable>"
            )


        callable_fallback_630v5.append(
            (
                name_630v5,
                sig_630v5,
            )
        )


callable_fallback_630v5 = sorted(
    callable_fallback_630v5
)


# =============================================================================
# 4) LOCATOR STATE
# =============================================================================

FD004_FULL_EXECUTION_AUTHORITY_LOCATOR_COMPLETED_630V5 = True

FD004_CORRUPTION_EXECUTED_630V5 = False

FD004_MODEL_INFERENCE_PERFORMED_630V5 = False

FD004_MC_EXECUTION_PERFORMED_630V5 = False

FD004_DEV_TARGETS_ERRORS_USED_630V5 = False

FD004_THRESHOLD_SELECTED_630V5 = False

FD004_POLICY_SELECTED_630V5 = False

FD004_TEST_CONTENTS_READ_630V5 = False

FD004_RUL_CONTENTS_READ_630V5 = False


all_checks_passed_630v5 = True


# =============================================================================
# 5) REPORT
# =============================================================================

print(
    "=" * 176
)

print(
    "STEP 6.30V-5 — FD004 FULL-EXECUTION BRIDGE + "
    "MANIFEST AUTHORITY LOCATOR"
)

print(
    "=" * 176
)


print(
    "Corruption executed              :",
    False,
)

print(
    "Model inference performed        :",
    False,
)

print(
    "MC execution performed           :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- EXPLICITLY WHITELISTED EXECUTION AUTHORITIES ---"
)

print(
    locator_df_630v5.to_string(
        index=False
    )
)


print(
    "\n--- NAMES-ONLY FALLBACK: 630T / 630U / 630V2 ---"
)

for name_630v5 in fallback_names_630v5:

    print(
        name_630v5
    )


print(
    "\n--- CALLABLE-ONLY BRIDGE FALLBACK ---"
)

for (
    name_630v5,
    sig_630v5,
) in callable_fallback_630v5:

    print(
        f"{name_630v5}{sig_630v5}"
    )


print(
    "\nPASS — only metadata and names of already-resident "
    "FD004 corruption-reference, admissibility-manifest, and "
    "smoke-bridge authorities were inspected. "
    "No corruption, inference, MC, DEV scoring, threshold, "
    "TEST, or RUL operation occurred."
)

In [ ]:
# STEP 6.30V-6
# FD004 COMPLETE 381-SCENARIO LABEL-FREE EXECUTION-CONTRACT FREEZE
#
# Contract / fingerprint step only.
# NO corruption.
# NO deterministic inference.
# NO MC execution.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v5", False)):
    raise RuntimeError(
        "STEP 6.30V-5 must pass before STEP 6.30V-6."
    )

if not bool(globals().get("all_checks_passed_630v4r2", False)):
    raise RuntimeError(
        "STEP 6.30V-4R2 must remain passed before STEP 6.30V-6."
    )

if not bool(globals().get("all_checks_passed_630v2", False)):
    raise RuntimeError(
        "STEP 6.30V-2 smoke bridge must remain passed."
    )


required_names_630v6 = (
    "FD004_CORRUPTION_MANIFEST_630S",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U",
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U",
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U",
    "FD004_CORRUPTION_MANIFEST_ANNOTATED_630U",
    "FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U",
    "FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_SHA256_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T",
    "FD004_REGIME_SENSOR_CORRUPTION_STATS_630T",
    "FD004_REGIME_SENSOR_CORRUPTION_STATS_SHA256_630T",
    "FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T",
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
    "FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2",
    "FD004_TRAINED_MODEL_SHA256_630J",
    "FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4R2",
)

missing_630v6 = [
    name_630v6
    for name_630v6 in required_names_630v6
    if name_630v6 not in globals()
]

if missing_630v6:
    raise RuntimeError(
        "Missing required STEP 6.30V-6 authority object(s): "
        + ", ".join(missing_630v6)
    )


# =============================================================================
# 1) VERIFY MANIFEST STRUCTURE / ORDER
# =============================================================================

manifest_630v6 = FD004_CORRUPTION_EXECUTION_MANIFEST_630U

if not isinstance(manifest_630v6, pd.DataFrame):
    raise RuntimeError(
        "FD004 execution manifest is not a DataFrame."
    )

if manifest_630v6.shape[0] != 381:
    raise RuntimeError(
        f"Expected 381 executable scenarios, found {manifest_630v6.shape[0]}."
    )


required_manifest_cols_630v6 = (
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
)

missing_cols_630v6 = [
    col_630v6
    for col_630v6 in required_manifest_cols_630v6
    if col_630v6 not in manifest_630v6.columns
]

if missing_cols_630v6:
    raise RuntimeError(
        "Execution manifest is missing original frozen-manifest columns: "
        + ", ".join(missing_cols_630v6)
    )


scenario_numbers_630v6 = manifest_630v6[
    "scenario_number"
].to_numpy(dtype=np.int64)

if len(np.unique(scenario_numbers_630v6)) != 381:
    raise RuntimeError(
        "Execution-manifest scenario numbers are not unique."
    )

if not np.all(
    scenario_numbers_630v6[:-1]
    <
    scenario_numbers_630v6[1:]
):
    raise RuntimeError(
        "Execution manifest is not in strictly ascending frozen scenario order."
    )


scenario_ids_630v6 = manifest_630v6[
    "scenario_id"
].astype(str).to_numpy()

if len(np.unique(scenario_ids_630v6)) != 381:
    raise RuntimeError(
        "Execution-manifest scenario IDs are not unique."
    )


expected_families_630v6 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

observed_families_630v6 = tuple(
    manifest_630v6["family"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if observed_families_630v6 != expected_families_630v6:
    raise RuntimeError(
        "Execution-manifest family order differs from the frozen design: "
        f"{observed_families_630v6}"
    )


family_counts_630v6 = (
    manifest_630v6["family"]
    .astype(str)
    .value_counts(sort=False)
    .reindex(expected_families_630v6)
    .astype(int)
)

expected_family_counts_630v6 = {
    "bias": 102,
    "noise": 51,
    "drift": 102,
    "dropout": 63,
    "stuck": 63,
}

if family_counts_630v6.to_dict() != expected_family_counts_630v6:
    raise RuntimeError(
        "Executable family counts differ from the frozen admissibility result: "
        f"{family_counts_630v6.to_dict()}"
    )


if int(FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U.shape[0]) != 60:
    raise RuntimeError(
        "Expected exactly 60 structurally inadmissible scenarios."
    )

if int(FD004_CORRUPTION_MANIFEST_630S.shape[0]) != 441:
    raise RuntimeError(
        "Original scientific manifest must remain 441 rows."
    )


# =============================================================================
# 2) LOCAL IMMUTABILITY DIGESTS
# =============================================================================

def _df_digest_630v6(df_630v6):

    h_630v6 = hashlib.sha256()

    h_630v6.update(
        repr(tuple(df_630v6.columns)).encode("utf-8")
    )

    h_630v6.update(
        repr(tuple(str(x) for x in df_630v6.dtypes)).encode("utf-8")
    )

    row_hashes_630v6 = pd.util.hash_pandas_object(
        df_630v6,
        index=True,
    ).to_numpy(dtype=np.uint64)

    h_630v6.update(
        np.ascontiguousarray(
            row_hashes_630v6
        ).tobytes()
    )

    return h_630v6.hexdigest()


def _canonicalize_630v6(value_630v6):

    if value_630v6 is None:
        return None

    if isinstance(
        value_630v6,
        (str, int, float, bool),
    ):
        return value_630v6

    if isinstance(
        value_630v6,
        np.generic,
    ):
        return value_630v6.item()

    if isinstance(
        value_630v6,
        dict,
    ):
        return {
            str(key_630v6): _canonicalize_630v6(
                value_630v6[key_630v6]
            )
            for key_630v6 in sorted(
                value_630v6.keys(),
                key=lambda z_630v6: str(z_630v6),
            )
        }

    if isinstance(
        value_630v6,
        (list, tuple),
    ):
        return [
            _canonicalize_630v6(item_630v6)
            for item_630v6 in value_630v6
        ]

    if isinstance(
        value_630v6,
        np.ndarray,
    ):
        arr_630v6 = np.ascontiguousarray(
            value_630v6
        )

        return {
            "type": "ndarray",
            "dtype": str(arr_630v6.dtype),
            "shape": list(arr_630v6.shape),
            "sha256": hashlib.sha256(
                arr_630v6.tobytes()
            ).hexdigest(),
        }

    if isinstance(
        value_630v6,
        pd.DataFrame,
    ):
        return {
            "type": "DataFrame",
            "shape": list(value_630v6.shape),
            "digest": _df_digest_630v6(
                value_630v6
            ),
        }

    return {
        "type": type(value_630v6).__name__,
        "repr": repr(value_630v6),
    }


FD004_EXECUTION_MANIFEST_LOCAL_SHA256_630V6 = (
    _df_digest_630v6(
        manifest_630v6
    )
)

FD004_OPERATOR_AUTHORITY_LOCAL_SHA256_630V6 = (
    hashlib.sha256(
        json.dumps(
            _canonicalize_630v6(
                FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2
            ),
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 3) FREEZE FAMILY-BLOCK EXECUTION PLAN
# =============================================================================

block_rows_630v6 = []

for block_index_630v6, family_630v6 in enumerate(
    expected_families_630v6,
    start=1,
):

    block_df_630v6 = manifest_630v6.loc[
        manifest_630v6["family"].astype(str)
        ==
        family_630v6
    ]

    block_rows_630v6.append(
        {
            "block_index":
                block_index_630v6,

            "family":
                family_630v6,

            "scenario_count":
                int(block_df_630v6.shape[0]),

            "first_scenario_number":
                int(
                    block_df_630v6[
                        "scenario_number"
                    ].iloc[0]
                ),

            "last_scenario_number":
                int(
                    block_df_630v6[
                        "scenario_number"
                    ].iloc[-1]
                ),
        }
    )


FD004_EXECUTION_BLOCK_PLAN_630V6 = pd.DataFrame(
    block_rows_630v6
)

FD004_EXECUTION_BLOCK_PLAN_630V6.set_index(
    "block_index",
    inplace=True,
)


# =============================================================================
# 4) FREEZE COMPLETE LABEL-FREE EXECUTION CONTRACT
# =============================================================================

FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6 = {

    "dataset":
        "FD004",

    "stage":
        "6.30V-6",

    "scientific_manifest_rows":
        441,

    "executable_scenarios":
        381,

    "structurally_inadmissible_scenarios":
        60,

    "execution_manifest_authority":
        "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",

    "execution_manifest_frozen_sha256":
        str(
            FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U
        ),

    "execution_manifest_local_sha256":
        FD004_EXECUTION_MANIFEST_LOCAL_SHA256_630V6,

    "annotated_manifest_frozen_sha256":
        str(
            FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U
        ),

    "inadmissible_manifest_frozen_sha256":
        str(
            FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U
        ),

    "admissibility_contract_sha256":
        str(
            FD004_CORRUPTION_ADMISSIBILITY_CONTRACT_SHA256_630U
        ),

    "corruption_reference_contract_sha256":
        str(
            FD004_CORRUPTION_REFERENCE_CONTRACT_SHA256_630T
        ),

    "corruption_scale_reference_sha256":
        str(
            FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T
        ),

    "dropout_imputation_reference_sha256":
        str(
            FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T
        ),

    "regime_sensor_reference_sha256":
        str(
            FD004_REGIME_SENSOR_CORRUPTION_STATS_SHA256_630T
        ),

    "historical_operator_authority_object":
        "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",

    "historical_operator_authority_local_sha256":
        FD004_OPERATOR_AUTHORITY_LOCAL_SHA256_630V6,

    "preprocessing_bridge":
        "normalize_fd004_raw_630v2",

    "window_bridge":
        "build_windows_fd004_630v2",

    "smoke_bridge_contract_sha256":
        str(
            FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2
        ),

    "model_authority":
        "MODEL_FD004_630I",

    "trained_model_sha256":
        str(
            FD004_TRAINED_MODEL_SHA256_630J
        ),

    "deterministic_eval_batch_size":
        512,

    "mc_helper":
        "fd004_mc_dropout_predict_630v4r2",

    "mc_passes":
        50,

    "mc_batch_size":
        512,

    "mc_seed":
        42,

    "mc_std_ddof":
        0,

    "reliability_helper":
        "apply_fd004_final_reliability_chain_630v4r2",

    "reliability_engine_contract_sha256":
        str(
            FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4R2
        ),

    "retained_per_scenario_arrays": [
        "deterministic_prediction",
        "mc_std",
        "final_selective_risk",
    ],

    "discard_full_mc_matrix_after_hash":
        True,

    "execution_order":
        list(
            expected_families_630v6
        ),

    "block_counts":
        {
            key_630v6:
                int(value_630v6)

            for key_630v6, value_630v6
            in family_counts_630v6.items()
        },

    "dev_targets_or_errors_used_during_execution":
        False,

    "threshold_or_policy_selected_during_execution":
        False,

    "test_used":
        False,

    "official_rul_used":
        False,
}


FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6 = (
    hashlib.sha256(
        json.dumps(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_LOCKED_630V6 = True

FD004_FULL_CORRUPTION_EXECUTED_630V6 = False

FD004_DEV_TARGETS_ERRORS_USED_630V6 = False

FD004_THRESHOLD_SELECTED_630V6 = False

FD004_POLICY_SELECTED_630V6 = False

FD004_TEST_CONTENTS_READ_630V6 = False

FD004_RUL_CONTENTS_READ_630V6 = False

all_checks_passed_630v6 = True


# =============================================================================
# 5) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-6 — FD004 COMPLETE 381-SCENARIO "
    "LABEL-FREE EXECUTION-CONTRACT FREEZE"
)

print("=" * 176)


print(
    "Corruption executed              :",
    False,
)

print(
    "Model inference performed        :",
    False,
)

print(
    "MC execution performed           :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- ARRANGED EXECUTION AUTHORITIES ---"
)

print(
    "Scientific manifest              : "
    "FD004_CORRUPTION_MANIFEST_630S (441 rows)"
)

print(
    "Executable manifest              : "
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U (381 rows)"
)

print(
    "Inadmissible manifest            : "
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U (60 rows)"
)

print(
    "Corruption scale reference       : "
    "FD004_CORRUPTION_SCALE_REFERENCE_630T"
)

print(
    "Dropout-imputation reference     : "
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T"
)

print(
    "Regime/sensor reference stats    : "
    "FD004_REGIME_SENSOR_CORRUPTION_STATS_630T"
)

print(
    "Normalization bridge             : "
    "normalize_fd004_raw_630v2"
)

print(
    "Window bridge                    : "
    "build_windows_fd004_630v2"
)

print(
    "Resident trained model           : "
    "MODEL_FD004_630I"
)

print(
    "MC engine                        : "
    "fd004_mc_dropout_predict_630v4r2"
)

print(
    "Reliability engine               : "
    "apply_fd004_final_reliability_chain_630v4r2"
)


print(
    "\n--- FROZEN FAMILY BLOCK PLAN ---"
)

print(
    FD004_EXECUTION_BLOCK_PLAN_630V6.to_string()
)


print(
    "\n--- RETENTION POLICY ---"
)

print(
    "Retain per scenario              : "
    "prediction + MC STD + final risk"
)

print(
    "Full 50-pass MC matrix           : "
    "hash then discard"
)

print(
    "DEV outcome/error use            : False"
)

print(
    "Threshold/policy selection       : False"
)


print(
    "\n--- CONTRACT FINGERPRINTS ---"
)

print(
    "Execution manifest frozen SHA-256:",
    FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U,
)

print(
    "Execution manifest local SHA-256 :",
    FD004_EXECUTION_MANIFEST_LOCAL_SHA256_630V6,
)

print(
    "Operator authority local SHA-256 :",
    FD004_OPERATOR_AUTHORITY_LOCAL_SHA256_630V6,
)

print(
    "Smoke bridge contract SHA-256    :",
    FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2,
)

print(
    "Reliability-engine SHA-256       :",
    FD004_RELIABILITY_ENGINE_CONTRACT_SHA256_630V4R2,
)

print(
    "Full execution contract SHA-256  :",
    FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6,
)


print(
    "\nPASS — the complete FD004 label-free corruption execution "
    "path is now prospectively bound and frozen for exactly "
    "381 admissible scenarios. The original 441-row scientific "
    "manifest and the 60 structurally inadmissible scenarios "
    "remain preserved. No corruption or outcome scoring occurred."
)

print(
    "Next: instantiate/verify the exact historical five-family "
    "corruption operator against the already-passed smoke scenario, "
    "then execute the frozen family blocks one at a time."
)

In [ ]:
# STEP 6.30V-7
# FD004 HISTORICAL FIVE-FAMILY CORRUPTION-OPERATOR SEMANTICS AUTHORITY AUDIT
#
# Source/evidence inspection only.
# NO corruption execution.
# NO inference.
# NO MC.
# NO DEV labels/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import pandas as pd
import hashlib
import json

# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v6", False)):
    raise RuntimeError(
        "STEP 6.30V-6 must pass before STEP 6.30V-7."
    )

required_630v7 = (
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",
    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_SCENARIO_NUMBER_630V2",
    "FD004_SMOKE_BIAS_AMOUNT_630V2",
    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
    "FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2",
)

missing_630v7 = [
    name_630v7
    for name_630v7 in required_630v7
    if name_630v7 not in globals()
]

if missing_630v7:
    raise RuntimeError(
        "Missing STEP 6.30V-7 authority object(s): "
        + ", ".join(missing_630v7)
    )

# =============================================================================
# 1) SAFE TEXT EXTRACTION FROM THE ALREADY-RECOVERED SOURCE-HITS OBJECT
# =============================================================================

source_hits_630v7 = (
    FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2
)

def _flatten_text_630v7(value, path="root", depth=0):
    out = []

    if depth > 6:
        return out

    if isinstance(value, str):
        out.append((path, value))
        return out

    if isinstance(value, dict):
        for key in sorted(value.keys(), key=lambda x: str(x)):
            out.extend(
                _flatten_text_630v7(
                    value[key],
                    path=f"{path}.{key}",
                    depth=depth + 1,
                )
            )
        return out

    if isinstance(value, (list, tuple)):
        for i, item in enumerate(value):
            out.extend(
                _flatten_text_630v7(
                    item,
                    path=f"{path}[{i}]",
                    depth=depth + 1,
                )
            )
        return out

    return out


all_text_fields_630v7 = _flatten_text_630v7(
    source_hits_630v7
)

operator_tokens_630v7 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
    "requested_fraction",
    "direction",
    "alpha",
    "seed",
    "normal",
    "choice",
    "random",
    "rng",
    "imput",
    "corrupt",
)

relevant_text_fields_630v7 = []

for path_630v7, text_630v7 in all_text_fields_630v7:
    lowered_630v7 = text_630v7.lower()

    if any(
        token_630v7 in lowered_630v7
        for token_630v7 in operator_tokens_630v7
    ):
        relevant_text_fields_630v7.append(
            (path_630v7, text_630v7)
        )

# De-duplicate exact text while preserving first occurrence.
deduped_relevant_630v7 = []
seen_text_630v7 = set()

for path_630v7, text_630v7 in relevant_text_fields_630v7:
    if text_630v7 in seen_text_630v7:
        continue

    seen_text_630v7.add(text_630v7)

    deduped_relevant_630v7.append(
        (path_630v7, text_630v7)
    )

# =============================================================================
# 2) EXISTING SMOKE-SCENARIO AUTHORITY
# =============================================================================

execution_manifest_630v7 = FD004_CORRUPTION_EXECUTION_MANIFEST_630U

smoke_number_630v7 = int(
    FD004_SMOKE_SCENARIO_NUMBER_630V2
)

smoke_id_630v7 = str(
    FD004_SMOKE_SCENARIO_ID_630V2
)

smoke_rows_630v7 = execution_manifest_630v7.loc[
    execution_manifest_630v7["scenario_number"].astype(int)
    == smoke_number_630v7
].copy()

if smoke_rows_630v7.shape[0] != 1:
    raise RuntimeError(
        "Smoke scenario was not uniquely located in the frozen execution manifest."
    )

smoke_row_630v7 = smoke_rows_630v7.iloc[0]

if str(smoke_row_630v7["scenario_id"]) != smoke_id_630v7:
    raise RuntimeError(
        "Smoke scenario ID/number mismatch."
    )

smoke_summary_630v7 = {
    "scenario_id": str(smoke_row_630v7["scenario_id"]),
    "scenario_number": int(smoke_row_630v7["scenario_number"]),
    "family": str(smoke_row_630v7["family"]),
    "sensor": str(smoke_row_630v7["sensor"]),
    "severity": str(smoke_row_630v7["severity"]),
    "alpha": (
        None
        if pd.isna(smoke_row_630v7["alpha"])
        else float(smoke_row_630v7["alpha"])
    ),
    "direction": (
        None
        if pd.isna(smoke_row_630v7["direction"])
        else float(smoke_row_630v7["direction"])
    ),
    "requested_fraction": (
        None
        if pd.isna(smoke_row_630v7["requested_fraction"])
        else float(smoke_row_630v7["requested_fraction"])
    ),
    "seed": (
        None
        if pd.isna(smoke_row_630v7["seed"])
        else int(smoke_row_630v7["seed"])
    ),
    "smoke_bias_amount": float(
        FD004_SMOKE_BIAS_AMOUNT_630V2
    ),
}

# =============================================================================
# 3) SOURCE-AUTHORITY FINGERPRINT
# =============================================================================

def _canonical_source_630v7(value):
    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(k): _canonical_source_630v7(value[k])
            for k in sorted(value.keys(), key=lambda x: str(x))
        }

    if isinstance(value, (list, tuple)):
        return [
            _canonical_source_630v7(item)
            for item in value
        ]

    return {
        "type": type(value).__name__,
        "repr": repr(value),
    }


FD004_OPERATOR_SOURCE_AUTHORITY_AUDIT_SHA256_630V7 = (
    hashlib.sha256(
        json.dumps(
            _canonical_source_630v7(
                source_hits_630v7
            ),
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

# =============================================================================
# 4) LOCK AUDIT STATE — NO OPERATOR YET
# =============================================================================

FD004_OPERATOR_SOURCE_TEXT_FIELD_COUNT_630V7 = int(
    len(all_text_fields_630v7)
)

FD004_OPERATOR_RELEVANT_SOURCE_FIELD_COUNT_630V7 = int(
    len(deduped_relevant_630v7)
)

FD004_HISTORICAL_OPERATOR_SEMANTICS_AUDITED_630V7 = True
FD004_HISTORICAL_OPERATOR_INSTANTIATED_630V7 = False
FD004_CORRUPTION_EXECUTED_630V7 = False
FD004_MODEL_INFERENCE_PERFORMED_630V7 = False
FD004_MC_EXECUTION_PERFORMED_630V7 = False
FD004_DEV_TARGETS_ERRORS_USED_630V7 = False
FD004_THRESHOLD_SELECTED_630V7 = False
FD004_POLICY_SELECTED_630V7 = False
FD004_TEST_CONTENTS_READ_630V7 = False
FD004_RUL_CONTENTS_READ_630V7 = False

all_checks_passed_630v7 = True

# =============================================================================
# 5) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-7 — FD004 HISTORICAL FIVE-FAMILY "
    "CORRUPTION-OPERATOR SEMANTICS AUTHORITY AUDIT"
)

print("=" * 176)

print("Corruption executed              :", False)
print("Model inference performed        :", False)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- SOURCE AUTHORITY ---")
print(
    "Authority object type            :",
    type(source_hits_630v7).__name__,
)
print(
    "Total nested text fields         :",
    FD004_OPERATOR_SOURCE_TEXT_FIELD_COUNT_630V7,
)
print(
    "Relevant unique source fields    :",
    FD004_OPERATOR_RELEVANT_SOURCE_FIELD_COUNT_630V7,
)
print(
    "Authority audit SHA-256          :",
    FD004_OPERATOR_SOURCE_AUTHORITY_AUDIT_SHA256_630V7,
)

print("\n--- EXISTING SMOKE AUTHORITY ---")

for key_630v7, value_630v7 in smoke_summary_630v7.items():
    print(
        f"{key_630v7:32s}: {value_630v7}"
    )

print(
    "Smoke X SHA-256                  :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "Smoke prediction SHA-256         :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)

print(
    "Smoke bridge contract SHA-256    :",
    FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2,
)

print(
    "\n--- RECOVERED HISTORICAL OPERATOR SOURCE FIELDS ---"
)

if not deduped_relevant_630v7:
    print(
        "<No text field containing operator-family tokens was found.>"
    )

else:
    for idx_630v7, (path_630v7, text_630v7) in enumerate(
        deduped_relevant_630v7,
        start=1,
    ):
        print(
            f"\n[{idx_630v7}] PATH: {path_630v7}"
        )
        print("-" * 120)
        print(text_630v7)

print(
    "\nPASS — the already-recovered historical corruption-operator "
    "authority and existing smoke scenario were audited without "
    "executing any corruption or model/reliability operation."
)

print(
    "Next: resolve the exact bias/noise/drift/dropout/stuck operator "
    "semantics from this authority, instantiate them once, and require "
    "exact smoke reproduction before block-1 execution."
)

In [ ]:
# STEP 6.30V-8
# FD004 HISTORICAL OPERATOR FREEZE + EXACT SMOKE REPRODUCTION LOCK
#
# Purpose:
#   1) freeze the five historical family authorities by exact function name
#   2) resolve executable bias semantics from the already-passed smoke evidence
#   3) reproduce the frozen smoke bridge exactly (X + deterministic prediction)
#   4) bind this verified operator layer for later block execution
#
# Safety:
#   - exactly one already-admissible smoke scenario is reconstructed
#   - no MC
#   - no DEV label/error use
#   - no threshold/policy
#   - no TEST/RUL

import numpy as np
import pandas as pd
import hashlib
import inspect
import json
import torch

# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v7", False)):
    raise RuntimeError(
        "STEP 6.30V-7 must pass before STEP 6.30V-8."
    )

required_630v8 = (
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_SCENARIO_NUMBER_630V2",
    "FD004_SMOKE_BIAS_AMOUNT_630V2",
)

missing_630v8 = [
    name_630v8
    for name_630v8 in required_630v8
    if name_630v8 not in globals()
]

if missing_630v8:
    raise RuntimeError(
        "Missing STEP 6.30V-8 requirement(s): "
        + ", ".join(missing_630v8)
    )

# =============================================================================
# 1) RESOLVE THE FIVE HISTORICAL FAMILY AUTHORITIES
# =============================================================================

source_hits_630v8 = FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2

def _extract_defined_functions_630v8(obj_630v8):
    names_630v8 = []

    def _walk_630v8(x_630v8):
        if isinstance(x_630v8, dict):
            for k_630v8, v_630v8 in x_630v8.items():
                if (
                    str(k_630v8) == "defined_functions"
                    and isinstance(v_630v8, (list, tuple))
                ):
                    for item_630v8 in v_630v8:
                        if isinstance(item_630v8, str):
                            names_630v8.append(item_630v8)
                else:
                    _walk_630v8(v_630v8)

        elif isinstance(x_630v8, (list, tuple)):
            for item_630v8 in x_630v8:
                _walk_630v8(item_630v8)

    _walk_630v8(obj_630v8)
    return list(dict.fromkeys(names_630v8))

defined_functions_630v8 = _extract_defined_functions_630v8(
    source_hits_630v8
)

expected_family_function_names_630v8 = {
    "bias": "apply_fd002_bias_628gc",
    "noise": "apply_fd002_noise_628gc",
    "drift": "apply_fd002_drift_628gc",
    "dropout": "apply_fd002_dropout_628gc",
    "stuck": "apply_fd002_stuck_628gc",
}

missing_names_630v8 = [
    fn_630v8
    for fn_630v8 in expected_family_function_names_630v8.values()
    if fn_630v8 not in defined_functions_630v8
]

if missing_names_630v8:
    raise RuntimeError(
        "Historical operator authority did not expose all five expected "
        "family function names. Missing: "
        + ", ".join(missing_names_630v8)
    )

resolved_operator_callables_630v8 = {}
resolved_operator_metadata_630v8 = {}

for family_630v8, fn_name_630v8 in expected_family_function_names_630v8.items():
    fn_630v8 = globals().get(fn_name_630v8, None)

    if fn_630v8 is None or not callable(fn_630v8):
        raise RuntimeError(
            f"Historical operator function '{fn_name_630v8}' "
            f"for family '{family_630v8}' is not callable in globals()."
        )

    try:
        signature_630v8 = str(inspect.signature(fn_630v8))
    except Exception:
        signature_630v8 = "<signature unavailable>"

    try:
        source_text_630v8 = inspect.getsource(fn_630v8)
    except Exception:
        source_text_630v8 = repr(fn_630v8)

    source_sha_630v8 = hashlib.sha256(
        source_text_630v8.encode("utf-8")
    ).hexdigest()

    resolved_operator_callables_630v8[family_630v8] = fn_630v8
    resolved_operator_metadata_630v8[family_630v8] = {
        "function_name": fn_name_630v8,
        "signature": signature_630v8,
        "source_sha256": source_sha_630v8,
    }

FD004_HISTORICAL_OPERATOR_FUNCTIONS_630V8 = resolved_operator_callables_630v8
FD004_HISTORICAL_OPERATOR_METADATA_630V8 = resolved_operator_metadata_630v8

# =============================================================================
# 2) LOCATE THE FROZEN SMOKE SCENARIO
# =============================================================================

manifest_630v8 = FD004_CORRUPTION_EXECUTION_MANIFEST_630U.copy()

smoke_id_630v8 = str(FD004_SMOKE_SCENARIO_ID_630V2)
smoke_number_630v8 = int(FD004_SMOKE_SCENARIO_NUMBER_630V2)

smoke_rows_630v8 = manifest_630v8.loc[
    manifest_630v8["scenario_number"].astype(int) == smoke_number_630v8
].copy()

if smoke_rows_630v8.shape[0] != 1:
    raise RuntimeError(
        "Frozen smoke scenario was not uniquely located in execution manifest."
    )

smoke_row_630v8 = smoke_rows_630v8.iloc[0]

if str(smoke_row_630v8["scenario_id"]) != smoke_id_630v8:
    raise RuntimeError(
        "Smoke scenario ID mismatch."
    )

if str(smoke_row_630v8["family"]) != "bias":
    raise RuntimeError(
        "STEP 6.30V-8 expects the already-passed smoke scenario to be a bias-family scenario."
    )

smoke_sensor_630v8 = str(smoke_row_630v8["sensor"])
smoke_alpha_630v8 = float(smoke_row_630v8["alpha"])
smoke_direction_630v8 = float(smoke_row_630v8["direction"])
smoke_severity_630v8 = str(smoke_row_630v8["severity"])

# =============================================================================
# 3) RESOLVE EXECUTABLE BIAS SEMANTICS FROM FROZEN SCALE REFERENCE
# =============================================================================

scale_ref_630v8 = FD004_CORRUPTION_SCALE_REFERENCE_630T.copy()

if "sensor" not in scale_ref_630v8.columns:
    raise RuntimeError(
        "FD004_CORRUPTION_SCALE_REFERENCE_630T must contain a 'sensor' column."
    )

scale_row_630v8 = scale_ref_630v8.loc[
    scale_ref_630v8["sensor"].astype(str) == smoke_sensor_630v8
].copy()

if scale_row_630v8.shape[0] != 1:
    raise RuntimeError(
        f"Could not uniquely locate scale reference row for sensor '{smoke_sensor_630v8}'."
    )

scale_row_630v8 = scale_row_630v8.iloc[0]

if "pooled_within_regime_std" not in scale_row_630v8.index:
    raise RuntimeError(
        "Scale reference must contain 'pooled_within_regime_std'."
    )

historical_pooled_scale_630v8 = float(
    scale_row_630v8["pooled_within_regime_std"]
)

if not np.isfinite(historical_pooled_scale_630v8):
    raise RuntimeError(
        "Historical pooled scale for smoke sensor is non-finite."
    )

smoke_bias_amount_rebuilt_630v8 = (
    smoke_direction_630v8
    * smoke_alpha_630v8
    * historical_pooled_scale_630v8
)

smoke_bias_amount_frozen_630v8 = float(
    FD004_SMOKE_BIAS_AMOUNT_630V2
)

if not np.isclose(
    smoke_bias_amount_rebuilt_630v8,
    smoke_bias_amount_frozen_630v8,
    rtol=0.0,
    atol=1e-15,
):
    raise RuntimeError(
        "Smoke bias amount rebuilt from frozen scale reference "
        "does not exactly match the frozen smoke amount."
    )

# =============================================================================
# 4) APPLY THE RESOLVED BIAS SEMANTICS TO RAW DEV DATA
#    (This is the exact executable semantic locked by smoke evidence.)
# =============================================================================

dev_raw_630v8 = FD004_DEV_RAW_630C.copy(deep=True)

if smoke_sensor_630v8 not in dev_raw_630v8.columns:
    raise RuntimeError(
        f"Smoke sensor column '{smoke_sensor_630v8}' not present in FD004_DEV_RAW_630C."
    )

smoke_corrupted_raw_630v8 = dev_raw_630v8.copy(deep=True)
smoke_corrupted_raw_630v8.loc[:, smoke_sensor_630v8] = (
    smoke_corrupted_raw_630v8[smoke_sensor_630v8].astype(np.float64)
    + smoke_bias_amount_rebuilt_630v8
)

# =============================================================================
# 5) REBUILD NORMALIZATION + WINDOWS EXACTLY THROUGH THE FROZEN BRIDGE
# =============================================================================

smoke_normalized_630v8 = normalize_fd004_raw_630v2(
    smoke_corrupted_raw_630v8,
    FD004_DEV_REGIME_IDS_630F,
)

smoke_X_630v8 = build_windows_fd004_630v2(
    smoke_normalized_630v8
)

smoke_X_630v8 = np.asarray(
    smoke_X_630v8,
    dtype=np.float32,
)

# =============================================================================
# 6) EXACT SMOKE-X REPRODUCTION CHECK
# =============================================================================

def _sha256_array_630v8(arr_630v8, dtype_630v8=None):
    arr_np_630v8 = np.asarray(arr_630v8)
    if dtype_630v8 is not None:
        arr_np_630v8 = arr_np_630v8.astype(dtype_630v8, copy=False)
    arr_np_630v8 = np.ascontiguousarray(arr_np_630v8)
    return hashlib.sha256(arr_np_630v8.tobytes()).hexdigest()

smoke_X_exact_630v8 = None
smoke_X_ref_sha_630v8 = None
smoke_X_max_abs_diff_630v8 = None

if "FD004_SMOKE_X_DEV_630V2" in globals():
    smoke_X_ref_630v8 = np.asarray(
        FD004_SMOKE_X_DEV_630V2,
        dtype=np.float32,
    )
    smoke_X_exact_630v8 = (
        smoke_X_ref_630v8.shape == smoke_X_630v8.shape
        and np.array_equal(smoke_X_ref_630v8, smoke_X_630v8)
    )
    smoke_X_max_abs_diff_630v8 = float(
        np.max(np.abs(smoke_X_ref_630v8 - smoke_X_630v8))
    )
    smoke_X_ref_sha_630v8 = _sha256_array_630v8(
        smoke_X_ref_630v8,
        dtype_630v8=np.float32,
    )
else:
    smoke_X_ref_sha_630v8 = str(FD004_SMOKE_X_SHA256_630V2)
    smoke_X_exact_630v8 = (
        _sha256_array_630v8(smoke_X_630v8, dtype_630v8=np.float32)
        == smoke_X_ref_sha_630v8
    )
    smoke_X_max_abs_diff_630v8 = None

if not bool(smoke_X_exact_630v8):
    raise RuntimeError(
        "Resolved historical bias semantics did not reproduce the frozen smoke X exactly."
    )

# =============================================================================
# 7) DETERMINISTIC SMOKE PREDICTION REPRODUCTION
# =============================================================================

model_630v8 = MODEL_FD004_630I
device_630v8 = FD004_DEVICE_630I

def _predict_deterministic_630v8(X_630v8, model_630v8, device_630v8, batch_size_630v8=512):
    X_630v8 = np.asarray(X_630v8, dtype=np.float32)
    outputs_630v8 = []

    model_630v8.eval()

    with torch.no_grad():
        for start_630v8 in range(0, X_630v8.shape[0], batch_size_630v8):
            end_630v8 = min(start_630v8 + batch_size_630v8, X_630v8.shape[0])

            xb_630v8 = torch.from_numpy(
                X_630v8[start_630v8:end_630v8]
            ).to(device_630v8)

            yb_630v8 = model_630v8(xb_630v8)

            if isinstance(yb_630v8, (tuple, list)):
                if len(yb_630v8) != 1:
                    raise RuntimeError(
                        "Deterministic model returned a tuple/list with unexpected length."
                    )
                yb_630v8 = yb_630v8[0]

            yb_630v8 = yb_630v8.detach().cpu().numpy().reshape(-1)
            outputs_630v8.append(yb_630v8)

    return np.concatenate(outputs_630v8, axis=0).astype(np.float32, copy=False)

smoke_pred_630v8 = _predict_deterministic_630v8(
    smoke_X_630v8,
    model_630v8=model_630v8,
    device_630v8=device_630v8,
    batch_size_630v8=512,
)

smoke_pred_exact_630v8 = None
smoke_pred_ref_sha_630v8 = None
smoke_pred_max_abs_diff_630v8 = None

if "FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2" in globals():
    smoke_pred_ref_630v8 = np.asarray(
        FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2,
        dtype=np.float32,
    ).reshape(-1)

    smoke_pred_exact_630v8 = (
        smoke_pred_ref_630v8.shape == smoke_pred_630v8.shape
        and np.array_equal(smoke_pred_ref_630v8, smoke_pred_630v8)
    )

    smoke_pred_max_abs_diff_630v8 = float(
        np.max(np.abs(smoke_pred_ref_630v8 - smoke_pred_630v8))
    )

    smoke_pred_ref_sha_630v8 = _sha256_array_630v8(
        smoke_pred_ref_630v8,
        dtype_630v8=np.float32,
    )
else:
    smoke_pred_ref_sha_630v8 = str(
        FD004_SMOKE_PREDICTION_SHA256_630V2
    )

    smoke_pred_exact_630v8 = (
        _sha256_array_630v8(smoke_pred_630v8, dtype_630v8=np.float32)
        == smoke_pred_ref_sha_630v8
    )

    smoke_pred_max_abs_diff_630v8 = None

if not bool(smoke_pred_exact_630v8):
    raise RuntimeError(
        "Resolved historical bias semantics reproduced smoke X but not the frozen smoke prediction."
    )

# =============================================================================
# 8) FREEZE THE VERIFIED OPERATOR / BRIDGE CONTRACT
# =============================================================================

FD004_RESOLVED_HISTORICAL_BIAS_SEMANTIC_630V8 = {
    "family": "bias",
    "sensor": smoke_sensor_630v8,
    "formula": "direction * alpha * pooled_within_regime_std",
    "pooled_within_regime_std": historical_pooled_scale_630v8,
    "alpha": smoke_alpha_630v8,
    "direction": smoke_direction_630v8,
    "reproduced_bias_amount": smoke_bias_amount_rebuilt_630v8,
}

FD004_HISTORICAL_OPERATOR_SMOKE_SCENARIO_630V8 = {
    "scenario_id": smoke_id_630v8,
    "scenario_number": smoke_number_630v8,
    "family": "bias",
    "sensor": smoke_sensor_630v8,
    "severity": smoke_severity_630v8,
}

FD004_HISTORICAL_OPERATOR_SMOKE_X_SHA256_630V8 = _sha256_array_630v8(
    smoke_X_630v8,
    dtype_630v8=np.float32,
)

FD004_HISTORICAL_OPERATOR_SMOKE_PRED_SHA256_630V8 = _sha256_array_630v8(
    smoke_pred_630v8,
    dtype_630v8=np.float32,
)

FD004_HISTORICAL_OPERATOR_FREEZE_CONTRACT_630V8 = {
    "resolved_function_names": {
        fam_630v8: meta_630v8["function_name"]
        for fam_630v8, meta_630v8 in FD004_HISTORICAL_OPERATOR_METADATA_630V8.items()
    },
    "resolved_function_signatures": {
        fam_630v8: meta_630v8["signature"]
        for fam_630v8, meta_630v8 in FD004_HISTORICAL_OPERATOR_METADATA_630V8.items()
    },
    "resolved_function_source_sha256": {
        fam_630v8: meta_630v8["source_sha256"]
        for fam_630v8, meta_630v8 in FD004_HISTORICAL_OPERATOR_METADATA_630V8.items()
    },
    "bias_semantic": FD004_RESOLVED_HISTORICAL_BIAS_SEMANTIC_630V8,
    "smoke_scenario": FD004_HISTORICAL_OPERATOR_SMOKE_SCENARIO_630V8,
    "smoke_x_sha256": FD004_HISTORICAL_OPERATOR_SMOKE_X_SHA256_630V8,
    "smoke_prediction_sha256": FD004_HISTORICAL_OPERATOR_SMOKE_PRED_SHA256_630V8,
    "smoke_bridge_contract_sha256": str(FD004_SMOKE_BRIDGE_CONTRACT_SHA256_630V2),
}

FD004_HISTORICAL_OPERATOR_FREEZE_CONTRACT_SHA256_630V8 = hashlib.sha256(
    json.dumps(
        FD004_HISTORICAL_OPERATOR_FREEZE_CONTRACT_630V8,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

FD004_HISTORICAL_OPERATOR_BOUND_630V8 = True
FD004_HISTORICAL_OPERATOR_SMOKE_VERIFIED_630V8 = True

FD004_CORRUPTION_EXECUTED_630V8 = False
FD004_MODEL_INFERENCE_PERFORMED_630V8 = True   # smoke deterministic reproduction only
FD004_MC_EXECUTION_PERFORMED_630V8 = False
FD004_DEV_TARGETS_ERRORS_USED_630V8 = False
FD004_THRESHOLD_SELECTED_630V8 = False
FD004_POLICY_SELECTED_630V8 = False
FD004_TEST_CONTENTS_READ_630V8 = False
FD004_RUL_CONTENTS_READ_630V8 = False

all_checks_passed_630v8 = True

# =============================================================================
# 9) REPORT
# =============================================================================

print("=" * 176)
print(
    "STEP 6.30V-8 — FD004 HISTORICAL OPERATOR FREEZE + EXACT SMOKE REPRODUCTION LOCK"
)
print("=" * 176)

print("Corruption executed              :", False)
print("Model inference performed        :", True)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- FROZEN HISTORICAL FAMILY AUTHORITIES ---")
for fam_630v8 in ("bias", "noise", "drift", "dropout", "stuck"):
    meta_630v8 = FD004_HISTORICAL_OPERATOR_METADATA_630V8[fam_630v8]
    print(f"{fam_630v8:10s}: {meta_630v8['function_name']}  {meta_630v8['signature']}")

print("\n--- RESOLVED EXECUTABLE BIAS SEMANTIC ---")
print("Smoke scenario ID                :", smoke_id_630v8)
print("Smoke scenario number            :", smoke_number_630v8)
print("Sensor                           :", smoke_sensor_630v8)
print("Severity / alpha / direction     :", (smoke_severity_630v8, smoke_alpha_630v8, smoke_direction_630v8))
print("Historical pooled scale          :", historical_pooled_scale_630v8)
print("Rebuilt bias amount              :", smoke_bias_amount_rebuilt_630v8)
print("Frozen smoke bias amount         :", smoke_bias_amount_frozen_630v8)

print("\n--- EXACT SMOKE REPRODUCTION ---")
print("Rebuilt smoke X shape            :", smoke_X_630v8.shape)
print("Smoke X exact                    :", bool(smoke_X_exact_630v8))
print("Smoke X SHA-256                  :", FD004_HISTORICAL_OPERATOR_SMOKE_X_SHA256_630V8)
if smoke_X_max_abs_diff_630v8 is not None:
    print("Smoke X max abs diff             :", smoke_X_max_abs_diff_630v8)

print("Rebuilt smoke prediction shape   :", smoke_pred_630v8.shape)
print("Smoke prediction exact           :", bool(smoke_pred_exact_630v8))
print("Smoke prediction SHA-256         :", FD004_HISTORICAL_OPERATOR_SMOKE_PRED_SHA256_630V8)
if smoke_pred_max_abs_diff_630v8 is not None:
    print("Smoke prediction max abs diff    :", smoke_pred_max_abs_diff_630v8)

print("\n--- CONTRACT FINGERPRINTS ---")
for fam_630v8 in ("bias", "noise", "drift", "dropout", "stuck"):
    print(
        f"{fam_630v8:10s} source SHA-256 : "
        f"{FD004_HISTORICAL_OPERATOR_METADATA_630V8[fam_630v8]['source_sha256']}"
    )

print(
    "Freeze contract SHA-256          :",
    FD004_HISTORICAL_OPERATOR_FREEZE_CONTRACT_SHA256_630V8,
)

print(
    "\nPASS — the historical five-family corruption authorities are now "
    "bound by exact function identity, and the executable bias-family "
    "semantics are verified end-to-end by exact smoke reproduction."
)

print(
    "Next: execute the frozen 381-scenario admissible manifest in the "
    "five locked family blocks (bias → noise → drift → dropout → stuck), "
    "retaining prediction + MC STD + final risk per scenario only."
)

In [ ]:
# STEP 6.30V-8R1
# FD004 HISTORICAL CORRUPTION-OPERATOR SOURCE-DEFINITION AUTHORITY LOCATOR
#
# Metadata / source-location inspection only.
# NO corruption.
# NO model inference.
# NO MC.
# NO DEV labels/errors.
# NO threshold/policy.
# NO TEST/RUL.

import numpy as np
import pandas as pd
import hashlib

# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

if not bool(globals().get("all_checks_passed_630v7", False)):
    raise RuntimeError(
        "STEP 6.30V-7 must pass before STEP 6.30V-8R1."
    )

if not bool(globals().get("all_checks_passed_630v6", False)):
    raise RuntimeError(
        "STEP 6.30V-6 must remain passed before STEP 6.30V-8R1."
    )

if bool(globals().get("FD004_HISTORICAL_OPERATOR_BOUND_630V8", False)):
    raise RuntimeError(
        "Unexpected state: failed STEP 6.30V-8 must not have bound an operator."
    )

# =============================================================================
# 1) EXACT RECOVERED AUTHORITY OBJECT
# =============================================================================

authority_name_630v8r1 = (
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2"
)

if authority_name_630v8r1 not in globals():
    raise RuntimeError(
        "Recovered historical corruption-operator source authority is missing."
    )

authority_630v8r1 = globals()[authority_name_630v8r1]

# =============================================================================
# 2) SAFE STRUCTURE WALK
# =============================================================================

tree_rows_630v8r1 = []

def _record_630v8r1(path_630v8r1, value_630v8r1, depth_630v8r1=0):
    if depth_630v8r1 > 6:
        return

    type_name_630v8r1 = type(value_630v8r1).__name__
    shape_or_len_630v8r1 = None
    preview_630v8r1 = None

    if isinstance(value_630v8r1, np.ndarray):
        shape_or_len_630v8r1 = tuple(value_630v8r1.shape)
        preview_630v8r1 = f"dtype={value_630v8r1.dtype}"

    elif isinstance(value_630v8r1, pd.DataFrame):
        shape_or_len_630v8r1 = tuple(value_630v8r1.shape)
        preview_630v8r1 = "columns=" + repr(list(value_630v8r1.columns))

    elif isinstance(value_630v8r1, pd.Series):
        shape_or_len_630v8r1 = tuple(value_630v8r1.shape)
        preview_630v8r1 = f"dtype={value_630v8r1.dtype}"

    elif isinstance(value_630v8r1, str):
        shape_or_len_630v8r1 = len(value_630v8r1)

        if len(value_630v8r1) <= 500:
            preview_630v8r1 = value_630v8r1
        else:
            preview_630v8r1 = (
                "<long text; "
                f"sha256={hashlib.sha256(value_630v8r1.encode('utf-8')).hexdigest()}, "
                f"prefix={value_630v8r1[:180]!r}>"
            )

    elif isinstance(value_630v8r1, dict):
        shape_or_len_630v8r1 = len(value_630v8r1)
        preview_630v8r1 = "keys=" + repr(
            [str(k_630v8r1) for k_630v8r1 in value_630v8r1.keys()]
        )

    elif isinstance(value_630v8r1, (list, tuple)):
        shape_or_len_630v8r1 = len(value_630v8r1)

    elif isinstance(value_630v8r1, (int, float, bool, np.generic)):
        preview_630v8r1 = repr(
            value_630v8r1.item()
            if isinstance(value_630v8r1, np.generic)
            else value_630v8r1
        )

    tree_rows_630v8r1.append(
        {
            "path": path_630v8r1,
            "type": type_name_630v8r1,
            "shape_or_len": shape_or_len_630v8r1,
            "safe_preview": preview_630v8r1,
        }
    )

    if isinstance(value_630v8r1, dict):
        for key_630v8r1, child_630v8r1 in value_630v8r1.items():
            _record_630v8r1(
                f"{path_630v8r1}.{key_630v8r1}",
                child_630v8r1,
                depth_630v8r1 + 1,
            )

    elif isinstance(value_630v8r1, (list, tuple)):
        for idx_630v8r1, child_630v8r1 in enumerate(value_630v8r1):
            _record_630v8r1(
                f"{path_630v8r1}[{idx_630v8r1}]",
                child_630v8r1,
                depth_630v8r1 + 1,
            )

_record_630v8r1(
    authority_name_630v8r1,
    authority_630v8r1,
)

authority_tree_df_630v8r1 = pd.DataFrame(
    tree_rows_630v8r1
)

# =============================================================================
# 3) NAMES-ONLY SEARCH FOR RELATED RECOVERY / SOURCE OBJECTS
# =============================================================================

related_names_630v8r1 = sorted(
    name_630v8r1
    for name_630v8r1 in globals().keys()
    if (
        "630TR2" in name_630v8r1.upper()
        or "HISTORICAL_CORRUPTION_OPERATOR" in name_630v8r1.upper()
        or "628GC" in name_630v8r1.upper()
        or "APPLY_FD002_BIAS" in name_630v8r1.upper()
        or "APPLY_FD002_NOISE" in name_630v8r1.upper()
        or "APPLY_FD002_DRIFT" in name_630v8r1.upper()
        or "APPLY_FD002_DROPOUT" in name_630v8r1.upper()
        or "APPLY_FD002_STUCK" in name_630v8r1.upper()
    )
)

related_rows_630v8r1 = []

for name_630v8r1 in related_names_630v8r1:
    obj_630v8r1 = globals()[name_630v8r1]

    if isinstance(obj_630v8r1, np.ndarray):
        shape_or_len_630v8r1 = tuple(obj_630v8r1.shape)
    elif isinstance(obj_630v8r1, (pd.DataFrame, pd.Series)):
        shape_or_len_630v8r1 = tuple(obj_630v8r1.shape)
    elif isinstance(obj_630v8r1, (str, list, tuple, dict)):
        shape_or_len_630v8r1 = len(obj_630v8r1)
    else:
        shape_or_len_630v8r1 = None

    related_rows_630v8r1.append(
        {
            "name": name_630v8r1,
            "type": type(obj_630v8r1).__name__,
            "shape_or_len": shape_or_len_630v8r1,
            "callable": bool(callable(obj_630v8r1)),
        }
    )

related_df_630v8r1 = pd.DataFrame(
    related_rows_630v8r1
)

# =============================================================================
# 4) LOCATOR STATE
# =============================================================================

FD004_HISTORICAL_OPERATOR_SOURCE_DEFINITION_LOCATOR_COMPLETED_630V8R1 = True
FD004_CORRUPTION_EXECUTED_630V8R1 = False
FD004_MODEL_INFERENCE_PERFORMED_630V8R1 = False
FD004_MC_EXECUTION_PERFORMED_630V8R1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V8R1 = False
FD004_THRESHOLD_SELECTED_630V8R1 = False
FD004_POLICY_SELECTED_630V8R1 = False
FD004_TEST_CONTENTS_READ_630V8R1 = False
FD004_RUL_CONTENTS_READ_630V8R1 = False

all_checks_passed_630v8r1 = True

# =============================================================================
# 5) REPORT
# =============================================================================

print("=" * 176)
print(
    "STEP 6.30V-8R1 — FD004 HISTORICAL CORRUPTION-OPERATOR "
    "SOURCE-DEFINITION AUTHORITY LOCATOR"
)
print("=" * 176)

print("Prior 6.30V-8 failure cause      : historical names recovered, functions not resident")
print("Corruption executed              :", False)
print("Model inference performed        :", False)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- RECOVERED SOURCE-AUTHORITY STRUCTURE ---")
print(
    authority_tree_df_630v8r1.to_string(
        index=False
    )
)

print("\n--- RELATED RESIDENT NAMES / METADATA ONLY ---")
if related_df_630v8r1.empty:
    print("<none>")
else:
    print(
        related_df_630v8r1.to_string(
            index=False
        )
    )

print(
    "\nPASS — only the structure/metadata of the already-recovered "
    "historical operator-source authority and related resident names "
    "were inspected. No experimental operation occurred."
)

print(
    "Next: use the exact recovered notebook path/cell/source authority "
    "revealed above to reconstruct the five historical operators; do "
    "not infer their semantics from function names."
)

In [ ]:
# STEP 6.30V-8R2
# FD004 EXACT HISTORICAL OPERATOR + DISPATCHER SOURCE EXTRACTION / FINGERPRINT LOCK
#
# SOURCE INSPECTION ONLY.
# No historical cell execution.
# No function definition execution.
# No corruption.
# No model inference / MC.
# No DEV outcome use.
# No threshold/policy.
# No TEST/RUL.

import hashlib
import ast

if not bool(globals().get("all_checks_passed_630v8r1", False)):
    raise RuntimeError(
        "STEP 6.30V-8R1 must pass before STEP 6.30V-8R2."
    )

required_630v8r2 = (
    "cells_630tr2",
    "FD002_NOTEBOOK_PATH_630TR2",
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
)

missing_630v8r2 = [
    name_630v8r2
    for name_630v8r2 in required_630v8r2
    if name_630v8r2 not in globals()
]

if missing_630v8r2:
    raise RuntimeError(
        "Missing STEP 6.30V-8R2 authority object(s): "
        + ", ".join(missing_630v8r2)
    )

cells_authority_630v8r2 = cells_630tr2

if not isinstance(cells_authority_630v8r2, list):
    raise RuntimeError(
        "cells_630tr2 is not the expected notebook-cell list."
    )

if len(cells_authority_630v8r2) <= 27:
    raise RuntimeError(
        "Historical notebook cell list does not contain cells 25 and 27."
    )

expected_authorities_630v8r2 = {
    25: {
        "source_sha256":
            "74dc1af52e2359a74191cb3121915b89bb42f4522e1add88e39b1c82e68ced2e",
        "defined_functions": (
            "apply_fd002_bias_628gc",
            "apply_fd002_noise_628gc",
            "apply_fd002_drift_628gc",
            "apply_fd002_dropout_628gc",
            "apply_fd002_stuck_628gc",
        ),
    },
    27: {
        "source_sha256":
            "51055c9790966038aea53e473ca46cb97db18784f351cb3cc105894e9e514dfe",
        "defined_functions": (
            "apply_manifest_scenario_628ha",
        ),
    },
}

extracted_sources_630v8r2 = {}
extracted_metadata_630v8r2 = {}

def _cell_source_text_630v8r2(cell_630v8r2):
    if not isinstance(cell_630v8r2, dict):
        raise RuntimeError(
            "Historical notebook cell is not a dictionary."
        )

    source_630v8r2 = cell_630v8r2.get("source", None)

    if source_630v8r2 is None:
        raise RuntimeError(
            "Historical notebook cell has no source field."
        )

    if isinstance(source_630v8r2, list):
        source_text_630v8r2 = "".join(
            str(x_630v8r2)
            for x_630v8r2 in source_630v8r2
        )
    elif isinstance(source_630v8r2, str):
        source_text_630v8r2 = source_630v8r2
    else:
        raise RuntimeError(
            "Historical cell source has unexpected type: "
            f"{type(source_630v8r2).__name__}"
        )

    return source_text_630v8r2

for cell_index_630v8r2, authority_630v8r2 in expected_authorities_630v8r2.items():

    cell_630v8r2 = cells_authority_630v8r2[cell_index_630v8r2]

    cell_type_630v8r2 = str(
        cell_630v8r2.get("cell_type", "")
    )

    if cell_type_630v8r2 != "code":
        raise RuntimeError(
            f"Historical cell {cell_index_630v8r2} is not a code cell."
        )

    source_text_630v8r2 = _cell_source_text_630v8r2(
        cell_630v8r2
    )

    source_sha_630v8r2 = hashlib.sha256(
        source_text_630v8r2.encode("utf-8")
    ).hexdigest()

    if source_sha_630v8r2 != authority_630v8r2["source_sha256"]:
        raise RuntimeError(
            f"Historical cell {cell_index_630v8r2} source SHA-256 changed.\n"
            f"Expected: {authority_630v8r2['source_sha256']}\n"
            f"Observed: {source_sha_630v8r2}"
        )

    try:
        tree_630v8r2 = ast.parse(
            source_text_630v8r2
        )
    except SyntaxError as exc_630v8r2:
        raise RuntimeError(
            f"Historical cell {cell_index_630v8r2} source cannot be parsed."
        ) from exc_630v8r2

    function_nodes_630v8r2 = [
        node_630v8r2
        for node_630v8r2 in tree_630v8r2.body
        if isinstance(
            node_630v8r2,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        )
    ]

    defined_here_630v8r2 = tuple(
        node_630v8r2.name
        for node_630v8r2 in function_nodes_630v8r2
    )

    expected_functions_630v8r2 = authority_630v8r2[
        "defined_functions"
    ]

    for fn_name_630v8r2 in expected_functions_630v8r2:
        if fn_name_630v8r2 not in defined_here_630v8r2:
            raise RuntimeError(
                f"Expected historical function '{fn_name_630v8r2}' "
                f"not found in exact cell {cell_index_630v8r2} source."
            )

    function_signatures_630v8r2 = {}

    for node_630v8r2 in function_nodes_630v8r2:
        if node_630v8r2.name not in expected_functions_630v8r2:
            continue

        args_630v8r2 = []

        for arg_630v8r2 in node_630v8r2.args.posonlyargs:
            args_630v8r2.append(arg_630v8r2.arg)

        if node_630v8r2.args.posonlyargs:
            args_630v8r2.append("/")

        for arg_630v8r2 in node_630v8r2.args.args:
            args_630v8r2.append(arg_630v8r2.arg)

        if node_630v8r2.args.vararg is not None:
            args_630v8r2.append(
                "*" + node_630v8r2.args.vararg.arg
            )
        elif node_630v8r2.args.kwonlyargs:
            args_630v8r2.append("*")

        for arg_630v8r2 in node_630v8r2.args.kwonlyargs:
            args_630v8r2.append(arg_630v8r2.arg)

        if node_630v8r2.args.kwarg is not None:
            args_630v8r2.append(
                "**" + node_630v8r2.args.kwarg.arg
            )

        function_signatures_630v8r2[
            node_630v8r2.name
        ] = "(" + ", ".join(args_630v8r2) + ")"

    extracted_sources_630v8r2[
        cell_index_630v8r2
    ] = source_text_630v8r2

    extracted_metadata_630v8r2[
        cell_index_630v8r2
    ] = {
        "source_sha256": source_sha_630v8r2,
        "defined_functions": defined_here_630v8r2,
        "expected_function_signatures":
            function_signatures_630v8r2,
        "source_char_count":
            len(source_text_630v8r2),
        "source_line_count":
            len(source_text_630v8r2.splitlines()),
    }

FD004_HISTORICAL_OPERATOR_SOURCE_CELL_630V8R2 = (
    extracted_sources_630v8r2[25]
)

FD004_HISTORICAL_DISPATCHER_SOURCE_CELL_630V8R2 = (
    extracted_sources_630v8r2[27]
)

FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2 = (
    extracted_metadata_630v8r2[25]["source_sha256"]
)

FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2 = (
    extracted_metadata_630v8r2[27]["source_sha256"]
)

FD004_HISTORICAL_OPERATOR_SOURCE_EXTRACTED_630V8R2 = True
FD004_HISTORICAL_SOURCE_EXECUTED_630V8R2 = False
FD004_CORRUPTION_EXECUTED_630V8R2 = False
FD004_MODEL_INFERENCE_PERFORMED_630V8R2 = False
FD004_MC_EXECUTION_PERFORMED_630V8R2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V8R2 = False
FD004_THRESHOLD_SELECTED_630V8R2 = False
FD004_POLICY_SELECTED_630V8R2 = False
FD004_TEST_CONTENTS_READ_630V8R2 = False
FD004_RUL_CONTENTS_READ_630V8R2 = False

all_checks_passed_630v8r2 = True

print("=" * 176)
print(
    "STEP 6.30V-8R2 — FD004 EXACT HISTORICAL OPERATOR + "
    "DISPATCHER SOURCE EXTRACTION / FINGERPRINT LOCK"
)
print("=" * 176)

print(
    "Historical notebook             :",
    FD002_NOTEBOOK_PATH_630TR2,
)
print("Historical cells executed        :", False)
print("Corruption executed              :", False)
print("Model inference performed        :", False)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- EXACT AUTHORITY METADATA ---")

for cell_index_630v8r2 in (25, 27):
    meta_630v8r2 = extracted_metadata_630v8r2[
        cell_index_630v8r2
    ]

    print(
        f"\nHistorical cell {cell_index_630v8r2}"
    )
    print(
        "  source SHA-256                :",
        meta_630v8r2["source_sha256"],
    )
    print(
        "  source chars / lines          :",
        (
            meta_630v8r2["source_char_count"],
            meta_630v8r2["source_line_count"],
        ),
    )
    print(
        "  defined functions             :",
        meta_630v8r2["defined_functions"],
    )
    print(
        "  expected signatures           :",
        meta_630v8r2["expected_function_signatures"],
    )

print(
    "\n--- EXACT HISTORICAL CELL 25 SOURCE: FIVE FAMILY OPERATORS ---"
)
print(FD004_HISTORICAL_OPERATOR_SOURCE_CELL_630V8R2)

print(
    "\n--- EXACT HISTORICAL CELL 27 SOURCE: MANIFEST DISPATCHER ---"
)
print(FD004_HISTORICAL_DISPATCHER_SOURCE_CELL_630V8R2)

print(
    "\nPASS — exact source text for historical cells 25 and 27 "
    "was extracted from the recovered FD002 notebook and matched "
    "the previously locked SHA-256 fingerprints. Nothing was executed."
)

print(
    "Next: inspect these exact definitions, isolate only the required "
    "operator functions and their explicit dependencies, and reconstruct "
    "them without executing unrelated historical notebook statements."
)

In [ ]:
# ============================================================
# STEP 6.30V-9
# FD004 EXACT HISTORICAL FIVE-FAMILY OPERATOR RECONSTRUCTION
# + EXISTING-SMOKE EXACT-REPRODUCTION LOCK
#
# IMPORTANT
# ---------
# - reconstructs only the five historical corruption operators
# - reconstructs the historical manifest dispatcher semantics
# - uses FD004 FIT-only frozen references
# - replays ONLY the already-established smoke corruption bridge
# - performs NO new model inference
# - performs NO MC-dropout
# - uses NO DEV outcomes/errors
# - selects NO threshold/policy
# - reads NO TEST / official RUL
# - does NOT run the 381-scenario experiment yet
# ============================================================

import numpy as np
import pandas as pd
import hashlib
import json


# ============================================================
# 1. PRECONDITIONS
# ============================================================

required_630v9 = (
    "all_checks_passed_630v8r2",

    "FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2",
    "FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2",

    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",

    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",

    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_SENSOR_COLS_630E",

    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",

    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_SCENARIO_NUMBER_630V2",
    "FD004_SMOKE_BIAS_AMOUNT_630V2",

    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_SMOKE_NORMALIZED_630V2",
    "FD004_SMOKE_X_DEV_630V2",

    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",

    "FD004_FULL_EXECUTION_CONTRACT_SHA256_630V6",
)


missing_630v9 = [
    name_630v9
    for name_630v9 in required_630v9
    if name_630v9 not in globals()
]


if missing_630v9:

    raise RuntimeError(
        "Missing STEP 6.30V-9 authority object(s): "
        + ", ".join(
            missing_630v9
        )
    )


if not bool(
    all_checks_passed_630v8r2
):

    raise RuntimeError(
        "STEP 6.30V-8R2 did not pass."
    )


# ============================================================
# 2. EXACT HISTORICAL SOURCE AUTHORITY
# ============================================================

EXPECTED_OPERATOR_SOURCE_SHA256_630V9 = (
    "74dc1af52e2359a74191cb3121915b89"
    "bb42f4522e1add88e39b1c82e68ced2e"
)


EXPECTED_DISPATCHER_SOURCE_SHA256_630V9 = (
    "51055c9790966038aea53e473ca46cb9"
    "7db18784f351cb3cc105894e9e514dfe"
)


if (
    FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2
    !=
    EXPECTED_OPERATOR_SOURCE_SHA256_630V9
):

    raise RuntimeError(
        "Historical operator-source authority changed."
    )


if (
    FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2
    !=
    EXPECTED_DISPATCHER_SOURCE_SHA256_630V9
):

    raise RuntimeError(
        "Historical dispatcher-source authority changed."
    )


# ============================================================
# 3. FREEZE FD004 REFERENCE LOOKUPS
# ============================================================

sensor_cols_630v9 = tuple(
    str(x_630v9)
    for x_630v9
    in FD004_SENSOR_COLS_630E
)


if len(
    sensor_cols_630v9
) != 21:

    raise RuntimeError(
        "Expected exactly 21 FD004 physical sensors."
    )


scale_ref_630v9 = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T
    .copy()
)


dropout_ref_630v9 = (
    FD004_DROPOUT_IMPUTATION_REFERENCE_630T
    .copy()
)


for col_630v9 in (
    "sensor",
    "pooled_within_regime_std",
):

    if col_630v9 not in scale_ref_630v9.columns:

        raise RuntimeError(
            "Required corruption-scale column missing: "
            + col_630v9
        )


for col_630v9 in (
    "regime_id",
    "sensor",
    "mean",
):

    if col_630v9 not in dropout_ref_630v9.columns:

        raise RuntimeError(
            "Required dropout-reference column missing: "
            + col_630v9
        )


SENSOR_SCALE_630V9 = {

    str(
        row_630v9["sensor"]
    ):
        float(
            row_630v9[
                "pooled_within_regime_std"
            ]
        )

    for _, row_630v9
    in scale_ref_630v9.iterrows()
}


DROPOUT_MEAN_630V9 = {

    (
        int(
            row_630v9["regime_id"]
        ),
        str(
            row_630v9["sensor"]
        ),
    ):
        float(
            row_630v9["mean"]
        )

    for _, row_630v9
    in dropout_ref_630v9.iterrows()
}


if (
    set(
        SENSOR_SCALE_630V9
    )
    !=
    set(
        sensor_cols_630v9
    )
):

    raise RuntimeError(
        "FD004 corruption-scale sensor coverage "
        "is not exact."
    )


expected_dropout_keys_630v9 = {

    (
        regime_630v9,
        sensor_630v9,
    )

    for regime_630v9
    in range(6)

    for sensor_630v9
    in sensor_cols_630v9
}


if (
    set(
        DROPOUT_MEAN_630V9
    )
    !=
    expected_dropout_keys_630v9
):

    raise RuntimeError(
        "FD004 dropout-imputation reference does not "
        "cover all 6 x 21 regime/sensor pairs exactly."
    )


# ============================================================
# 4. COMMON SENSOR VALIDATION
# ============================================================

def validate_sensor_fd004_630v9(
    df,
    sensor,
):

    if sensor not in sensor_cols_630v9:

        raise ValueError(
            f"{sensor} is not one of the "
            "21 frozen FD004 physical sensors."
        )


    if sensor not in df.columns:

        raise ValueError(
            f"{sensor} is missing from dataframe."
        )


# ============================================================
# 5. BIAS
#
# historical semantics:
#
# corrupted =
#     clean
#     + direction * alpha * FIT_scale(sensor)
# ============================================================

def apply_fd004_bias_630v9(
    df,
    sensor,
    alpha,
    direction,
):

    validate_sensor_fd004_630v9(
        df,
        sensor,
    )


    if direction not in (
        -1,
        1,
    ):

        raise ValueError(
            "direction must be -1 or +1."
        )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    scale = float(
        SENSOR_SCALE_630V9[
            sensor
        ]
    )


    if (
        not np.isfinite(
            scale
        )
        or
        scale <= 0.0
    ):

        raise ValueError(
            "Historical magnitude scale is unusable "
            f"for {sensor}: {scale}"
        )


    bias_amount = float(
        direction
        *
        alpha
        *
        scale
    )


    corrupted = (
        df.copy()
    )


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        bias_amount
    )


    return (
        corrupted,
        bias_amount,
    )


# ============================================================
# 6. NOISE
#
# historical semantics:
#
# rng = np.random.default_rng(seed)
# base_noise ~ N(0,1)
#
# noise =
#     base_noise
#     * alpha
#     * FIT_scale(sensor)
#
# Same seed => same base realization across severities.
# ============================================================

def apply_fd004_noise_630v9(
    df,
    sensor,
    alpha,
    seed=42,
):

    validate_sensor_fd004_630v9(
        df,
        sensor,
    )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    scale = float(
        SENSOR_SCALE_630V9[
            sensor
        ]
    )


    if (
        not np.isfinite(
            scale
        )
        or
        scale <= 0.0
    ):

        raise ValueError(
            "Historical magnitude scale is unusable "
            f"for {sensor}: {scale}"
        )


    target_std = float(
        alpha
        *
        scale
    )


    rng_630v9 = np.random.default_rng(
        int(
            seed
        )
    )


    base_noise_630v9 = (
        rng_630v9.normal(
            loc=0.0,
            scale=1.0,
            size=len(
                df
            ),
        )
    )


    noise_630v9 = (
        base_noise_630v9
        *
        target_std
    )


    corrupted = (
        df.copy()
    )


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        noise_630v9
    )


    return (
        corrupted,
        noise_630v9,
        target_std,
    )


# ============================================================
# 7. DRIFT
#
# historical semantics:
#
# independently for each engine:
#
# first cycle -> 0
# last cycle  -> direction * alpha * FIT_scale(sensor)
#
# linear interpolation between.
# ============================================================

def apply_fd004_drift_630v9(
    df,
    sensor,
    alpha,
    direction,
):

    validate_sensor_fd004_630v9(
        df,
        sensor,
    )


    if direction not in (
        -1,
        1,
    ):

        raise ValueError(
            "direction must be -1 or +1."
        )


    alpha = float(
        alpha
    )


    if (
        not np.isfinite(
            alpha
        )
        or
        alpha <= 0.0
    ):

        raise ValueError(
            "alpha must be positive and finite."
        )


    scale = float(
        SENSOR_SCALE_630V9[
            sensor
        ]
    )


    if (
        not np.isfinite(
            scale
        )
        or
        scale <= 0.0
    ):

        raise ValueError(
            "Historical magnitude scale is unusable "
            f"for {sensor}: {scale}"
        )


    final_drift_630v9 = float(
        direction
        *
        alpha
        *
        scale
    )


    corrupted = (
        df.copy()
    )


    drift_values_630v9 = np.zeros(
        len(
            corrupted
        ),
        dtype=np.float64,
    )


    for _, group_630v9 in (
        corrupted
        .groupby(
            "unit_id",
            sort=False,
        )
    ):

        group_sorted_630v9 = (
            group_630v9
            .sort_values(
                "cycle"
            )
        )


        positions_630v9 = (
            group_sorted_630v9
            .index
            .to_numpy(
                dtype=np.int64
            )
        )


        n_rows_630v9 = len(
            positions_630v9
        )


        if n_rows_630v9 == 1:

            engine_drift_630v9 = np.array(
                [
                    final_drift_630v9
                ],
                dtype=np.float64,
            )

        else:

            engine_drift_630v9 = np.linspace(
                0.0,
                final_drift_630v9,
                num=n_rows_630v9,
                endpoint=True,
                dtype=np.float64,
            )


        drift_values_630v9[
            positions_630v9
        ] = engine_drift_630v9


    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        drift_values_630v9
    )


    return (
        corrupted,
        drift_values_630v9,
        final_drift_630v9,
    )


# ============================================================
# 8. DROPOUT
#
# historical semantics:
#
# n_drop = round(rate * N)
# rng = default_rng(seed)
# random_order = permutation(N)
# drop_positions = first n_drop positions
#
# missing values then replaced by frozen FIT-only
# regime-specific sensor mean.
# ============================================================

def apply_fd004_dropout_630v9(
    df,
    sensor,
    dropout_rate,
    seed=42,
):

    validate_sensor_fd004_630v9(
        df,
        sensor,
    )


    dropout_rate = float(
        dropout_rate
    )


    if not (
        0.0
        <=
        dropout_rate
        <
        1.0
    ):

        raise ValueError(
            "dropout_rate must satisfy "
            "0 <= rate < 1."
        )


    if "regime_id" not in df.columns:

        raise ValueError(
            "regime_id is required for "
            "condition-aware imputation."
        )


    corrupted = (
        df.copy()
    )


    n_rows_630v9 = len(
        corrupted
    )


    n_drop_630v9 = int(
        round(
            dropout_rate
            *
            n_rows_630v9
        )
    )


    rng_630v9 = np.random.default_rng(
        int(
            seed
        )
    )


    random_order_630v9 = (
        rng_630v9.permutation(
            n_rows_630v9
        )
    )


    drop_positions_630v9 = (
        random_order_630v9[
            :n_drop_630v9
        ]
    )


    drop_mask_630v9 = np.zeros(
        n_rows_630v9,
        dtype=bool,
    )


    drop_mask_630v9[
        drop_positions_630v9
    ] = True


    sensor_col_pos_630v9 = (
        corrupted.columns
        .get_loc(
            sensor
        )
    )


    corrupted.iloc[
        drop_positions_630v9,
        sensor_col_pos_630v9,
    ] = np.nan


    regime_values_630v9 = (
        corrupted[
            "regime_id"
        ]
        .to_numpy()
    )


    imputation_values_630v9 = np.empty(
        n_rows_630v9,
        dtype=np.float64,
    )


    for position_630v9 in range(
        n_rows_630v9
    ):

        regime_630v9 = int(
            regime_values_630v9[
                position_630v9
            ]
        )


        key_630v9 = (
            regime_630v9,
            sensor,
        )


        if key_630v9 not in DROPOUT_MEAN_630V9:

            raise RuntimeError(
                "Missing FIT dropout mean for "
                f"regime={regime_630v9}, "
                f"sensor={sensor}."
            )


        imputation_values_630v9[
            position_630v9
        ] = (
            DROPOUT_MEAN_630V9[
                key_630v9
            ]
        )


    missing_positions_630v9 = (
        np.flatnonzero(
            drop_mask_630v9
        )
    )


    corrupted.iloc[
        missing_positions_630v9,
        sensor_col_pos_630v9,
    ] = (
        imputation_values_630v9[
            missing_positions_630v9
        ]
    )


    return (
        corrupted,
        drop_mask_630v9,
        imputation_values_630v9,
    )


# ============================================================
# 9. STUCK SENSOR
#
# historical semantics:
#
# independently for each engine:
#
# n_affected = round(fraction * trajectory length)
#
# constrained to:
#     >= 1 affected row
#     >= 1 valid row before onset
#
# terminal suffix frozen to final pre-onset value.
# ============================================================

def apply_fd004_stuck_630v9(
    df,
    sensor,
    affected_fraction,
):

    validate_sensor_fd004_630v9(
        df,
        sensor,
    )


    affected_fraction = float(
        affected_fraction
    )


    if not (
        0.0
        <
        affected_fraction
        <
        1.0
    ):

        raise ValueError(
            "affected_fraction must satisfy "
            "0 < fraction < 1."
        )


    corrupted = (
        df.copy()
    )


    affected_mask_630v9 = np.zeros(
        len(
            corrupted
        ),
        dtype=bool,
    )


    records_630v9 = []


    for unit_id_630v9, group_630v9 in (
        corrupted
        .groupby(
            "unit_id",
            sort=False,
        )
    ):

        group_sorted_630v9 = (
            group_630v9
            .sort_values(
                "cycle"
            )
        )


        positions_630v9 = (
            group_sorted_630v9
            .index
            .to_numpy(
                dtype=np.int64
            )
        )


        n_rows_630v9 = len(
            positions_630v9
        )


        if n_rows_630v9 < 2:

            raise RuntimeError(
                "Stuck corruption requires at least "
                "two cycles per engine."
            )


        n_affected_630v9 = int(
            round(
                affected_fraction
                *
                n_rows_630v9
            )
        )


        n_affected_630v9 = max(
            1,
            n_affected_630v9,
        )


        n_affected_630v9 = min(
            n_affected_630v9,
            n_rows_630v9 - 1,
        )


        onset_position_630v9 = (
            n_rows_630v9
            -
            n_affected_630v9
        )


        source_position_630v9 = (
            positions_630v9[
                onset_position_630v9
                -
                1
            ]
        )


        affected_positions_630v9 = (
            positions_630v9[
                onset_position_630v9:
            ]
        )


        freeze_value_630v9 = float(
            corrupted.loc[
                source_position_630v9,
                sensor,
            ]
        )


        corrupted.loc[
            affected_positions_630v9,
            sensor,
        ] = freeze_value_630v9


        affected_mask_630v9[
            affected_positions_630v9
        ] = True


        records_630v9.append(
            {
                "unit_id":
                    unit_id_630v9,

                "trajectory_length":
                    n_rows_630v9,

                "requested_fraction":
                    affected_fraction,

                "affected_count":
                    n_affected_630v9,

                "actual_fraction":
                    (
                        n_affected_630v9
                        /
                        n_rows_630v9
                    ),

                "source_cycle":
                    float(
                        group_sorted_630v9
                        .iloc[
                            onset_position_630v9
                            -
                            1
                        ][
                            "cycle"
                        ]
                    ),

                "stuck_start_cycle":
                    float(
                        group_sorted_630v9
                        .iloc[
                            onset_position_630v9
                        ][
                            "cycle"
                        ]
                    ),

                "freeze_value":
                    freeze_value_630v9,
            }
        )


    stuck_info_630v9 = pd.DataFrame(
        records_630v9
    )


    return (
        corrupted,
        affected_mask_630v9,
        stuck_info_630v9,
    )


# ============================================================
# 10. FREEZE RECONSTRUCTED OPERATOR CONTRACT
# ============================================================

FD004_CORRUPTION_OPERATOR_CONTRACT_630V9 = {

    "bias":
        apply_fd004_bias_630v9,

    "noise":
        apply_fd004_noise_630v9,

    "drift":
        apply_fd004_drift_630v9,

    "dropout":
        apply_fd004_dropout_630v9,

    "stuck":
        apply_fd004_stuck_630v9,

    "sensor_scale":
        SENSOR_SCALE_630V9.copy(),

    "dropout_mean_reference":
        DROPOUT_MEAN_630V9.copy(),

    "historical_operator_source_sha256":
        EXPECTED_OPERATOR_SOURCE_SHA256_630V9,

    "historical_dispatcher_source_sha256":
        EXPECTED_DISPATCHER_SOURCE_SHA256_630V9,
}


# ============================================================
# 11. HISTORICAL MANIFEST DISPATCHER
# ============================================================

def apply_manifest_scenario_fd004_630v9(
    base_raw,
    scenario_row,
):

    family_630v9 = str(
        scenario_row[
            "family"
        ]
    )


    sensor_630v9 = str(
        scenario_row[
            "sensor"
        ]
    )


    contract_630v9 = (
        FD004_CORRUPTION_OPERATOR_CONTRACT_630V9
    )


    if family_630v9 == "bias":

        corrupted_630v9, _ = (
            contract_630v9[
                "bias"
            ](
                base_raw,
                sensor_630v9,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )


    elif family_630v9 == "noise":

        corrupted_630v9, _, _ = (
            contract_630v9[
                "noise"
            ](
                base_raw,
                sensor_630v9,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )


    elif family_630v9 == "drift":

        corrupted_630v9, _, _ = (
            contract_630v9[
                "drift"
            ](
                base_raw,
                sensor_630v9,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )


    elif family_630v9 == "dropout":

        corrupted_630v9, _, _ = (
            contract_630v9[
                "dropout"
            ](
                base_raw,
                sensor_630v9,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )


    elif family_630v9 == "stuck":

        corrupted_630v9, _, _ = (
            contract_630v9[
                "stuck"
            ](
                base_raw,
                sensor_630v9,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
            )
        )


    else:

        raise ValueError(
            "Unknown corruption family: "
            + family_630v9
        )


    return corrupted_630v9


# ============================================================
# 12. BUILD FD004 RAW CORRUPTION BASE
#
# historical dropout requires regime_id to travel with raw rows.
# ============================================================

FD004_DEV_CORRUPTION_BASE_630V9 = (
    FD004_DEV_RAW_630C
    .copy()
    .reset_index(
        drop=True
    )
)


regime_ids_630v9 = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)


if (
    len(
        regime_ids_630v9
    )
    !=
    len(
        FD004_DEV_CORRUPTION_BASE_630V9
    )
):

    raise RuntimeError(
        "DEV regime IDs are not row-aligned "
        "with FD004 DEV raw data."
    )


FD004_DEV_CORRUPTION_BASE_630V9[
    "regime_id"
] = (
    regime_ids_630v9
)


# ============================================================
# 13. VERIFY THAT EXECUTABLE MANIFEST STILL EXCLUDES
#     STRUCTURALLY NONUSABLE MAGNITUDE-SCALE CASES
# ============================================================

manifest_630v9 = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .copy()
    .reset_index(
        drop=True
    )
)


magnitude_families_630v9 = {
    "bias",
    "noise",
    "drift",
}


bad_exec_rows_630v9 = []


for idx_630v9, row_630v9 in (
    manifest_630v9.iterrows()
):

    family_630v9 = str(
        row_630v9[
            "family"
        ]
    )


    if family_630v9 not in magnitude_families_630v9:
        continue


    sensor_630v9 = str(
        row_630v9[
            "sensor"
        ]
    )


    scale_630v9 = float(
        SENSOR_SCALE_630V9[
            sensor_630v9
        ]
    )


    if (
        not np.isfinite(
            scale_630v9
        )
        or
        scale_630v9 <= 1e-12
    ):

        bad_exec_rows_630v9.append(
            (
                int(
                    row_630v9[
                        "scenario_number"
                    ]
                ),
                family_630v9,
                sensor_630v9,
                scale_630v9,
            )
        )


if bad_exec_rows_630v9:

    raise RuntimeError(
        "Executable manifest contains a magnitude-based "
        "scenario with structurally nonusable FIT scale: "
        + str(
            bad_exec_rows_630v9[
                :10
            ]
        )
    )


# ============================================================
# 14. EXACT REPRODUCTION OF THE ALREADY-PASSED SMOKE SCENARIO
#
# No new model inference is necessary:
#
# if corrupted RAW -> normalized -> X is exact,
# the already-frozen smoke prediction remains authoritative.
# ============================================================

smoke_number_630v9 = int(
    FD004_SMOKE_SCENARIO_NUMBER_630V2
)


smoke_id_630v9 = str(
    FD004_SMOKE_SCENARIO_ID_630V2
)


smoke_matches_630v9 = (
    manifest_630v9.loc[
        manifest_630v9[
            "scenario_number"
        ].astype(
            int
        )
        ==
        smoke_number_630v9
    ]
    .copy()
)


if len(
    smoke_matches_630v9
) != 1:

    raise RuntimeError(
        "Smoke scenario number does not resolve "
        "to exactly one executable-manifest row."
    )


smoke_row_630v9 = (
    smoke_matches_630v9
    .iloc[
        0
    ]
)


if (
    str(
        smoke_row_630v9[
            "scenario_id"
        ]
    )
    !=
    smoke_id_630v9
):

    raise RuntimeError(
        "Smoke scenario ID / number authority mismatch."
    )


if (
    str(
        smoke_row_630v9[
            "family"
        ]
    )
    !=
    "bias"
):

    raise RuntimeError(
        "Frozen smoke scenario is unexpectedly "
        "not a bias scenario."
    )


(
    smoke_rebuilt_raw_630v9,
    smoke_bias_amount_630v9,
) = apply_fd004_bias_630v9(

    FD004_DEV_CORRUPTION_BASE_630V9,

    str(
        smoke_row_630v9[
            "sensor"
        ]
    ),

    float(
        smoke_row_630v9[
            "alpha"
        ]
    ),

    int(
        smoke_row_630v9[
            "direction"
        ]
    ),
)


smoke_bias_amount_exact_630v9 = bool(
    np.isclose(
        smoke_bias_amount_630v9,
        float(
            FD004_SMOKE_BIAS_AMOUNT_630V2
        ),
        atol=0.0,
        rtol=0.0,
    )
)


raw_cols_630v9 = list(
    FD004_DEV_RAW_630C.columns
)


frozen_smoke_raw_630v9 = (
    FD004_SMOKE_CORRUPTED_RAW_630V2
    .reset_index(
        drop=True
    )
)


smoke_raw_exact_630v9 = bool(
    np.array_equal(

        smoke_rebuilt_raw_630v9[
            raw_cols_630v9
        ].to_numpy(),

        frozen_smoke_raw_630v9[
            raw_cols_630v9
        ].to_numpy(),
    )
)


# ============================================================
# 15. NORMALIZATION REPRODUCTION
# ============================================================

smoke_rebuilt_norm_630v9 = (
    normalize_fd004_raw_630v2(

        smoke_rebuilt_raw_630v9[
            raw_cols_630v9
        ].copy(),

        regime_ids_630v9,
    )
)


frozen_smoke_norm_630v9 = (
    FD004_SMOKE_NORMALIZED_630V2
    .reset_index(
        drop=True
    )
)


common_norm_cols_630v9 = [

    col_630v9

    for col_630v9
    in frozen_smoke_norm_630v9.columns

    if col_630v9
    in smoke_rebuilt_norm_630v9.columns
]


if not common_norm_cols_630v9:

    raise RuntimeError(
        "No common normalized smoke columns found."
    )


smoke_normalized_exact_630v9 = bool(
    np.array_equal(

        smoke_rebuilt_norm_630v9[
            common_norm_cols_630v9
        ].to_numpy(),

        frozen_smoke_norm_630v9[
            common_norm_cols_630v9
        ].to_numpy(),
    )
)


# ============================================================
# 16. WINDOW REPRODUCTION
# ============================================================

smoke_rebuilt_X_630v9 = (
    build_windows_fd004_630v2(
        smoke_rebuilt_norm_630v9
    )
)


frozen_smoke_X_630v9 = np.asarray(
    FD004_SMOKE_X_DEV_630V2,
    dtype=np.float32,
)


smoke_X_exact_630v9 = bool(

    smoke_rebuilt_X_630v9.shape
    ==
    frozen_smoke_X_630v9.shape

    and

    np.array_equal(

        np.asarray(
            smoke_rebuilt_X_630v9,
            dtype=np.float32,
        ),

        frozen_smoke_X_630v9,
    )
)


if not (
    smoke_bias_amount_exact_630v9
    and
    smoke_raw_exact_630v9
    and
    smoke_normalized_exact_630v9
    and
    smoke_X_exact_630v9
):

    raise RuntimeError(
        "Reconstructed historical-semantics operator "
        "did not exactly reproduce the already-passed "
        "FD004 smoke bridge."
    )


# ============================================================
# 17. FREEZE SEMANTIC CONTRACT
# ============================================================

operator_semantics_630v9 = {

    "bias":
        (
            "clean + direction*alpha*"
            "FIT_pooled_within_regime_std"
        ),

    "noise":
        (
            "default_rng(seed).normal(0,1,N)*"
            "alpha*FIT_pooled_within_regime_std"
        ),

    "drift":
        (
            "per_engine_linear_0_to_"
            "direction*alpha*FIT_pooled_within_regime_std"
        ),

    "dropout":
        (
            "round(rate*N)_seeded_permutation_exact_count_"
            "then_FIT_regime_mean_imputation"
        ),

    "stuck":
        (
            "per_engine_terminal_round(fraction*length)_"
            "suffix_frozen_to_last_pre_onset_value"
        ),

    "dispatcher":
        (
            "family_specific_manifest_fields_"
            "exact_historical_routing"
        ),
}


FD004_CORRUPTION_OPERATOR_RECONSTRUCTION_CONTRACT_630V9 = {

    "historical_operator_source_sha256":
        EXPECTED_OPERATOR_SOURCE_SHA256_630V9,

    "historical_dispatcher_source_sha256":
        EXPECTED_DISPATCHER_SOURCE_SHA256_630V9,

    "full_execution_contract_sha256":
        str(
            FD004_FULL_EXECUTION_CONTRACT_SHA256_630V6
        ),

    "smoke_scenario_id":
        smoke_id_630v9,

    "smoke_scenario_number":
        smoke_number_630v9,

    "smoke_bias_amount_exact":
        smoke_bias_amount_exact_630v9,

    "smoke_raw_exact":
        smoke_raw_exact_630v9,

    "smoke_normalized_exact":
        smoke_normalized_exact_630v9,

    "smoke_X_exact":
        smoke_X_exact_630v9,

    "operator_semantics":
        operator_semantics_630v9,

    "full_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_OPERATOR_RECONSTRUCTION_CONTRACT_SHA256_630V9 = (
    hashlib.sha256(

        json.dumps(
            FD004_CORRUPTION_OPERATOR_RECONSTRUCTION_CONTRACT_630V9,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        ).encode(
            "utf-8"
        )

    ).hexdigest()
)


# ============================================================
# 18. LOCK FLAGS
# ============================================================

FD004_CORRUPTION_OPERATOR_CONTRACT_LOCKED_630V9 = True

FD004_HISTORICAL_OPERATOR_SEMANTICS_RESOLVED_630V9 = True

FD004_SMOKE_EXACTLY_REPRODUCED_630V9 = True

FD004_FULL_CORRUPTION_EXECUTED_630V9 = False

FD004_NEW_MODEL_INFERENCE_PERFORMED_630V9 = False

FD004_NEW_MC_EXECUTION_PERFORMED_630V9 = False

FD004_DEV_TARGETS_ERRORS_USED_630V9 = False

FD004_THRESHOLD_SELECTED_630V9 = False

FD004_POLICY_SELECTED_630V9 = False

FD004_TEST_CONTENTS_READ_630V9 = False

FD004_RUL_CONTENTS_READ_630V9 = False

all_checks_passed_630v9 = True


# ============================================================
# 19. REPORT
# ============================================================

print(
    "=" * 176
)

print(
    "STEP 6.30V-9 — FD004 EXACT HISTORICAL FIVE-FAMILY "
    "OPERATOR RECONSTRUCTION + SMOKE-REPRODUCTION LOCK"
)

print(
    "=" * 176
)


print(
    "Historical source executed       :",
    False,
)

print(
    "Five operators reconstructed     :",
    True,
)

print(
    "Dispatcher reconstructed         :",
    True,
)

print(
    "Full corruption run executed     :",
    False,
)

print(
    "New model inference performed    :",
    False,
)

print(
    "New MC execution performed       :",
    False,
)

print(
    "DEV targets/errors used          :",
    False,
)

print(
    "Threshold selected               :",
    False,
)

print(
    "Policy selected                  :",
    False,
)

print(
    "TEST contents read               :",
    False,
)

print(
    "Official RUL contents read       :",
    False,
)


print(
    "\n--- HISTORICAL AUTHORITY ---"
)

print(
    "Operator source SHA-256          :",
    EXPECTED_OPERATOR_SOURCE_SHA256_630V9,
)

print(
    "Dispatcher source SHA-256        :",
    EXPECTED_DISPATCHER_SOURCE_SHA256_630V9,
)


print(
    "\n--- RECONSTRUCTED FAMILY SEMANTICS ---"
)

for (
    family_630v9,
    semantics_630v9,
) in operator_semantics_630v9.items():

    print(
        f"{family_630v9:<12}: "
        f"{semantics_630v9}"
    )


print(
    "\n--- EXECUTABLE-MANIFEST SAFETY ---"
)

print(
    "Executable scenarios             :",
    len(
        manifest_630v9
    ),
)

print(
    "Nonusable magnitude rows present :",
    False,
)


print(
    "\n--- EXACT EXISTING-SMOKE REPRODUCTION ---"
)

print(
    "Scenario ID / number             :",
    (
        smoke_id_630v9,
        smoke_number_630v9,
    ),
)

print(
    "Family / sensor                  :",
    (
        str(
            smoke_row_630v9[
                "family"
            ]
        ),
        str(
            smoke_row_630v9[
                "sensor"
            ]
        ),
    ),
)

print(
    "Bias amount exact                :",
    smoke_bias_amount_exact_630v9,
)

print(
    "Corrupted raw exact              :",
    smoke_raw_exact_630v9,
)

print(
    "Normalized bridge exact          :",
    smoke_normalized_exact_630v9,
)

print(
    "Window tensor exact              :",
    smoke_X_exact_630v9,
)

print(
    "Frozen smoke X SHA-256           :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "Frozen smoke prediction SHA-256  :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)


print(
    "\n--- CONTRACT FINGERPRINT ---"
)

print(
    "Operator reconstruction contract SHA-256:",
    FD004_CORRUPTION_OPERATOR_RECONSTRUCTION_CONTRACT_SHA256_630V9,
)


print(
    "\nPASS — the five FD004 corruption operators and manifest "
    "dispatcher were reconstructed from the exact historical "
    "source semantics, and the already-passed FD004 smoke "
    "scenario was reproduced exactly through the "
    "corruption→normalization→window bridge."
)

print(
    "No new model inference, MC execution, DEV outcome use, "
    "threshold selection, TEST access, or official-RUL access occurred."
)

print(
    "Next: execute the prospectively frozen 381-scenario "
    "corruption experiment family block-by-family block, "
    "retaining only prediction, MC STD, final reliability risk, "
    "and the frozen evidence hashes specified by STEP 6.30V-6."
)

In [ ]:
# ============================================================
# STEP 6.30V-9R1
# FD004 STEP 6.30V-6 EXECUTION-CONTRACT AUTHORITY NAME LOCATOR
#
# PURPOSE
# -------
# Resolve the exact resident variable name(s) created by STEP 6.30V-6.
#
# NAMES / TYPE / SHAPE-LENGTH METADATA ONLY.
# NO numerical arrays inspected.
# NO corruption.
# NO inference.
# NO MC.
# NO DEV outcome/error access.
# NO threshold/policy selection.
# NO TEST / RUL access.
# ============================================================

print("=" * 176)
print(
    "STEP 6.30V-9R1 — FD004 STEP 6.30V-6 "
    "EXECUTION-CONTRACT AUTHORITY NAME LOCATOR"
)
print("=" * 176)

print("Corruption executed             :", False)
print("Model inference performed       :", False)
print("MC execution performed          :", False)
print("DEV targets/errors used         :", False)
print("Threshold selected              :", False)
print("Policy selected                 :", False)
print("TEST contents read              :", False)
print("Official RUL contents read      :", False)

candidate_names_630v9r1 = sorted(
    name_630v9r1
    for name_630v9r1 in globals()
    if (
        "630V6" in name_630v9r1.upper()
        or
        "630V_6" in name_630v9r1.upper()
    )
)

print("\n--- ALL RESIDENT STEP 6.30V-6 NAMES ---")

if not candidate_names_630v9r1:
    print("<none>")
else:
    for name_630v9r1 in candidate_names_630v9r1:
        obj_630v9r1 = globals()[name_630v9r1]

        type_name_630v9r1 = type(obj_630v9r1).__name__

        shape_or_len_630v9r1 = None

        if hasattr(obj_630v9r1, "shape"):
            try:
                shape_or_len_630v9r1 = tuple(obj_630v9r1.shape)
            except Exception:
                shape_or_len_630v9r1 = None

        if shape_or_len_630v9r1 is None:
            if isinstance(
                obj_630v9r1,
                (
                    str,
                    tuple,
                    list,
                    dict,
                    set,
                ),
            ):
                shape_or_len_630v9r1 = len(obj_630v9r1)

        print(
            f"{name_630v9r1:<72} "
            f"type={type_name_630v9r1:<18} "
            f"shape_or_len={shape_or_len_630v9r1}"
        )

contract_names_630v9r1 = tuple(
    name_630v9r1
    for name_630v9r1 in candidate_names_630v9r1
    if (
        "CONTRACT" in name_630v9r1.upper()
        or
        "MANIFEST" in name_630v9r1.upper()
        or
        "AUTHORITY" in name_630v9r1.upper()
        or
        "SHA256" in name_630v9r1.upper()
    )
)

print("\n--- CONTRACT / MANIFEST / AUTHORITY CANDIDATES ---")

if not contract_names_630v9r1:
    print("<none>")
else:
    for name_630v9r1 in contract_names_630v9r1:
        obj_630v9r1 = globals()[name_630v9r1]

        if isinstance(obj_630v9r1, str):
            safe_meta_630v9r1 = f"str_len={len(obj_630v9r1)}"
        elif isinstance(obj_630v9r1, dict):
            safe_meta_630v9r1 = (
                "dict_keys="
                + str(
                    sorted(
                        str(k_630v9r1)
                        for k_630v9r1
                        in obj_630v9r1.keys()
                    )
                )
            )
        elif hasattr(obj_630v9r1, "shape"):
            safe_meta_630v9r1 = (
                "shape="
                + str(
                    tuple(
                        obj_630v9r1.shape
                    )
                )
            )
        else:
            safe_meta_630v9r1 = (
                "type="
                + type(obj_630v9r1).__name__
            )

        print(
            f"{name_630v9r1:<72} "
            f"{safe_meta_630v9r1}"
        )

FD004_V6_AUTHORITY_NAME_LOCATOR_630V9R1 = tuple(
    candidate_names_630v9r1
)

FD004_V6_CONTRACT_NAME_CANDIDATES_630V9R1 = tuple(
    contract_names_630v9r1
)

all_checks_passed_630v9r1 = True

print(
    "\nPASS — only resident STEP 6.30V-6 object names and "
    "non-numerical metadata were inspected."
)
print(
    "No corruption, inference, MC, DEV scoring, threshold, "
    "TEST, or official-RUL operation occurred."
)

In [ ]:
# STEP 6.30V-9R2
# FD004 OPERATOR-RECONSTRUCTION STRICT PRE-FLIGHT AUTHORITY AUDIT
#
# NO corruption.
# NO model inference.
# NO MC.
# NO DEV outcome/error use.
# NO threshold/policy.
# NO TEST / RUL.

import inspect
import numpy as np
import pandas as pd

required_names_630v9r2 = (
    "all_checks_passed_630v8r2",
    "all_checks_passed_630v6",

    "FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2",
    "FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2",

    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",

    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",

    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_SENSOR_COLS_630E",

    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",

    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_SCENARIO_NUMBER_630V2",
    "FD004_SMOKE_BIAS_AMOUNT_630V2",

    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_SMOKE_NORMALIZED_630V2",
    "FD004_SMOKE_X_DEV_630V2",

    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
)

missing_630v9r2 = [
    name_630v9r2
    for name_630v9r2 in required_names_630v9r2
    if name_630v9r2 not in globals()
]

if missing_630v9r2:
    raise RuntimeError(
        "Missing required authority object(s): "
        + ", ".join(missing_630v9r2)
    )

if not bool(all_checks_passed_630v8r2):
    raise RuntimeError(
        "STEP 6.30V-8R2 is not in passed state."
    )

if not bool(all_checks_passed_630v6):
    raise RuntimeError(
        "STEP 6.30V-6 is not in passed state."
    )

EXPECTED_OPERATOR_SOURCE_SHA256_630V9R2 = (
    "74dc1af52e2359a74191cb3121915b89"
    "bb42f4522e1add88e39b1c82e68ced2e"
)

EXPECTED_DISPATCHER_SOURCE_SHA256_630V9R2 = (
    "51055c9790966038aea53e473ca46cb9"
    "7db18784f351cb3cc105894e9e514dfe"
)

if (
    FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2
    != EXPECTED_OPERATOR_SOURCE_SHA256_630V9R2
):
    raise RuntimeError(
        "Historical operator source fingerprint mismatch."
    )

if (
    FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2
    != EXPECTED_DISPATCHER_SOURCE_SHA256_630V9R2
):
    raise RuntimeError(
        "Historical dispatcher source fingerprint mismatch."
    )

manifest_630v9r2 = FD004_CORRUPTION_EXECUTION_MANIFEST_630U
scale_ref_630v9r2 = FD004_CORRUPTION_SCALE_REFERENCE_630T
drop_ref_630v9r2 = FD004_DROPOUT_IMPUTATION_REFERENCE_630T
dev_raw_630v9r2 = FD004_DEV_RAW_630C

dev_regime_ids_630v9r2 = np.asarray(
    FD004_DEV_REGIME_IDS_630F
)

sensor_cols_630v9r2 = tuple(
    FD004_SENSOR_COLS_630E
)

if (
    not isinstance(manifest_630v9r2, pd.DataFrame)
    or manifest_630v9r2.shape[0] != 381
):
    raise RuntimeError(
        "Executable manifest authority unexpected: "
        f"type={type(manifest_630v9r2).__name__}, "
        f"shape={getattr(manifest_630v9r2, 'shape', None)}"
    )

required_manifest_cols_630v9r2 = (
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
)

missing_manifest_cols_630v9r2 = [
    col_630v9r2
    for col_630v9r2 in required_manifest_cols_630v9r2
    if col_630v9r2 not in manifest_630v9r2.columns
]

if missing_manifest_cols_630v9r2:
    raise RuntimeError(
        "Executable manifest missing required columns: "
        + ", ".join(missing_manifest_cols_630v9r2)
    )

if not isinstance(scale_ref_630v9r2, pd.DataFrame):
    raise RuntimeError(
        "Corruption scale reference is not a DataFrame."
    )

if not isinstance(drop_ref_630v9r2, pd.DataFrame):
    raise RuntimeError(
        "Dropout imputation reference is not a DataFrame."
    )

scale_cols_630v9r2 = tuple(
    scale_ref_630v9r2.columns
)

drop_cols_630v9r2 = tuple(
    drop_ref_630v9r2.columns
)

if (
    not isinstance(dev_raw_630v9r2, pd.DataFrame)
    or dev_raw_630v9r2.shape[0] != 12274
):
    raise RuntimeError(
        "DEV raw authority unexpected: "
        f"type={type(dev_raw_630v9r2).__name__}, "
        f"shape={getattr(dev_raw_630v9r2, 'shape', None)}"
    )

if dev_regime_ids_630v9r2.shape != (len(dev_raw_630v9r2),):
    raise RuntimeError(
        "DEV regime-ID alignment mismatch: "
        f"{dev_regime_ids_630v9r2.shape} "
        f"vs {(len(dev_raw_630v9r2),)}"
    )

if len(sensor_cols_630v9r2) != 21:
    raise RuntimeError(
        "Expected exactly 21 physical sensors, found "
        f"{len(sensor_cols_630v9r2)}."
    )

if not callable(normalize_fd004_raw_630v2):
    raise RuntimeError(
        "normalize_fd004_raw_630v2 is not callable."
    )

if not callable(build_windows_fd004_630v2):
    raise RuntimeError(
        "build_windows_fd004_630v2 is not callable."
    )

normalize_sig_630v9r2 = str(
    inspect.signature(
        normalize_fd004_raw_630v2
    )
)

window_sig_630v9r2 = str(
    inspect.signature(
        build_windows_fd004_630v2
    )
)

smoke_raw_630v9r2 = FD004_SMOKE_CORRUPTED_RAW_630V2
smoke_norm_630v9r2 = FD004_SMOKE_NORMALIZED_630V2

smoke_x_630v9r2 = np.asarray(
    FD004_SMOKE_X_DEV_630V2
)

if not isinstance(smoke_raw_630v9r2, pd.DataFrame):
    raise RuntimeError(
        "Frozen smoke corrupted-raw object is not a DataFrame."
    )

if not isinstance(smoke_norm_630v9r2, pd.DataFrame):
    raise RuntimeError(
        "Frozen smoke normalized object is not a DataFrame."
    )

if smoke_x_630v9r2.shape != (10824, 30, 24):
    raise RuntimeError(
        "Frozen smoke X shape unexpected: "
        f"{smoke_x_630v9r2.shape}"
    )

FD004_OPERATOR_RECONSTRUCTION_PREFLIGHT_630V9R2 = {
    "operator_source_sha256":
        FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2,

    "dispatcher_source_sha256":
        FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2,

    "full_corruption_execution_contract_sha256":
        FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6,

    "manifest_shape":
        tuple(manifest_630v9r2.shape),

    "manifest_columns":
        tuple(manifest_630v9r2.columns),

    "scale_reference_shape":
        tuple(scale_ref_630v9r2.shape),

    "scale_reference_columns":
        scale_cols_630v9r2,

    "dropout_reference_shape":
        tuple(drop_ref_630v9r2.shape),

    "dropout_reference_columns":
        drop_cols_630v9r2,

    "dev_raw_shape":
        tuple(dev_raw_630v9r2.shape),

    "dev_regime_ids_shape":
        tuple(dev_regime_ids_630v9r2.shape),

    "sensor_count":
        len(sensor_cols_630v9r2),

    "normalize_signature":
        normalize_sig_630v9r2,

    "window_signature":
        window_sig_630v9r2,

    "smoke_raw_shape":
        tuple(smoke_raw_630v9r2.shape),

    "smoke_normalized_shape":
        tuple(smoke_norm_630v9r2.shape),

    "smoke_x_shape":
        tuple(smoke_x_630v9r2.shape),
}

FD004_OPERATOR_RECONSTRUCTION_PREFLIGHT_LOCKED_630V9R2 = True

FD004_CORRUPTION_EXECUTED_630V9R2 = False
FD004_MODEL_INFERENCE_PERFORMED_630V9R2 = False
FD004_MC_EXECUTION_PERFORMED_630V9R2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V9R2 = False
FD004_THRESHOLD_SELECTED_630V9R2 = False
FD004_POLICY_SELECTED_630V9R2 = False
FD004_TEST_CONTENTS_READ_630V9R2 = False
FD004_RUL_CONTENTS_READ_630V9R2 = False

all_checks_passed_630v9r2 = True

print("=" * 176)
print(
    "STEP 6.30V-9R2 — FD004 OPERATOR-RECONSTRUCTION "
    "STRICT PRE-FLIGHT AUTHORITY AUDIT"
)
print("=" * 176)

print("Corruption executed              :", False)
print("Model inference performed        :", False)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- EXACT FROZEN AUTHORITIES ---")

print(
    "Operator source SHA-256          :",
    FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2,
)

print(
    "Dispatcher source SHA-256        :",
    FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2,
)

print(
    "Full execution contract SHA-256  :",
    FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6,
)

print("\n--- EXECUTION MANIFEST ---")

print(
    "Shape                            :",
    tuple(manifest_630v9r2.shape),
)

print(
    "Required columns present         :",
    True,
)

print("Families / counts:")
print(
    manifest_630v9r2[
        "family"
    ].value_counts(
        sort=False
    ).to_string()
)

print("\n--- CORRUPTION REFERENCES ---")

print(
    "Scale reference shape            :",
    tuple(scale_ref_630v9r2.shape),
)

print(
    "Scale reference columns          :",
    list(scale_cols_630v9r2),
)

print(
    "Dropout reference shape          :",
    tuple(drop_ref_630v9r2.shape),
)

print(
    "Dropout reference columns        :",
    list(drop_cols_630v9r2),
)

print("\n--- DEV / BRIDGE AUTHORITIES ---")

print(
    "DEV raw shape                    :",
    tuple(dev_raw_630v9r2.shape),
)

print(
    "DEV regime-ID shape              :",
    tuple(dev_regime_ids_630v9r2.shape),
)

print(
    "Physical sensor count            :",
    len(sensor_cols_630v9r2),
)

print(
    "Normalization helper signature   :",
    normalize_sig_630v9r2,
)

print(
    "Window helper signature          :",
    window_sig_630v9r2,
)

print("\n--- FROZEN SMOKE AUTHORITIES ---")

print(
    "Scenario ID / number             :",
    (
        FD004_SMOKE_SCENARIO_ID_630V2,
        FD004_SMOKE_SCENARIO_NUMBER_630V2,
    ),
)

print(
    "Frozen bias amount               :",
    FD004_SMOKE_BIAS_AMOUNT_630V2,
)

print(
    "Corrupted raw shape              :",
    tuple(smoke_raw_630v9r2.shape),
)

print(
    "Normalized shape                 :",
    tuple(smoke_norm_630v9r2.shape),
)

print(
    "Window tensor shape              :",
    tuple(smoke_x_630v9r2.shape),
)

print(
    "Smoke X SHA-256                  :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "Smoke prediction SHA-256         :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)

print(
    "\nPASS — every authority required for the corrected "
    "historical-operator reconstruction is now explicitly "
    "resolved and structurally verified."
)

print(
    "No corruption, inference, MC, DEV scoring, threshold "
    "selection, TEST, or official-RUL operation occurred."
)

In [ ]:
# STEP 6.30V-9R3
# FD004 EXACT HISTORICAL BIAS-SMOKE RAW -> NORMALIZATION -> WINDOW REPRODUCTION LOCK
#
# This deliberately validates only the already-established smoke scenario.
# No full corruption experiment.
# No model inference.
# No MC.
# No DEV targets/errors.
# No threshold/policy.
# No TEST/RUL.

import numpy as np
import pandas as pd
import hashlib
import json

# ============================================================
# 0) PRECONDITIONS
# ============================================================

required_630v9r3 = (
    "all_checks_passed_630v9r2",
    "FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2",
    "FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_SCENARIO_NUMBER_630V2",
    "FD004_SMOKE_BIAS_AMOUNT_630V2",
    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_SMOKE_NORMALIZED_630V2",
    "FD004_SMOKE_X_DEV_630V2",
    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
)

missing_630v9r3 = [
    name_630v9r3
    for name_630v9r3 in required_630v9r3
    if name_630v9r3 not in globals()
]

if missing_630v9r3:
    raise RuntimeError(
        "Missing STEP 6.30V-9R3 authority object(s): "
        + ", ".join(missing_630v9r3)
    )

if not bool(all_checks_passed_630v9r2):
    raise RuntimeError(
        "STEP 6.30V-9R2 must pass before STEP 6.30V-9R3."
    )

# ============================================================
# 1) LOCK EXPECTED SOURCE FINGERPRINTS
# ============================================================

EXPECTED_OPERATOR_SOURCE_SHA256_630V9R3 = (
    "74dc1af52e2359a74191cb3121915b89"
    "bb42f4522e1add88e39b1c82e68ced2e"
)

EXPECTED_DISPATCHER_SOURCE_SHA256_630V9R3 = (
    "51055c9790966038aea53e473ca46cb9"
    "7db18784f351cb3cc105894e9e514dfe"
)

if (
    FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2
    != EXPECTED_OPERATOR_SOURCE_SHA256_630V9R3
):
    raise RuntimeError(
        "Historical five-family operator source fingerprint changed."
    )

if (
    FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2
    != EXPECTED_DISPATCHER_SOURCE_SHA256_630V9R3
):
    raise RuntimeError(
        "Historical manifest-dispatcher source fingerprint changed."
    )

# ============================================================
# 2) EXACT SMOKE MANIFEST AUTHORITY
# ============================================================

manifest_630v9r3 = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .copy()
    .reset_index(drop=True)
)

smoke_number_630v9r3 = int(
    FD004_SMOKE_SCENARIO_NUMBER_630V2
)

smoke_id_630v9r3 = str(
    FD004_SMOKE_SCENARIO_ID_630V2
)

smoke_rows_630v9r3 = manifest_630v9r3.loc[
    manifest_630v9r3["scenario_number"].astype(int)
    == smoke_number_630v9r3
].copy()

if len(smoke_rows_630v9r3) != 1:
    raise RuntimeError(
        "Frozen smoke scenario number does not resolve to exactly one "
        "executable-manifest row."
    )

smoke_row_630v9r3 = smoke_rows_630v9r3.iloc[0]

if str(smoke_row_630v9r3["scenario_id"]) != smoke_id_630v9r3:
    raise RuntimeError(
        "Frozen smoke scenario ID/number mismatch."
    )

if str(smoke_row_630v9r3["family"]) != "bias":
    raise RuntimeError(
        "Frozen smoke scenario is unexpectedly not a bias scenario."
    )

smoke_sensor_630v9r3 = str(
    smoke_row_630v9r3["sensor"]
)

smoke_alpha_630v9r3 = float(
    smoke_row_630v9r3["alpha"]
)

smoke_direction_630v9r3 = int(
    smoke_row_630v9r3["direction"]
)

if smoke_direction_630v9r3 not in (-1, 1):
    raise RuntimeError(
        "Frozen smoke bias direction is not -1 or +1."
    )

# ============================================================
# 3) EXACT FD004 FIT-ONLY MAGNITUDE SCALE
# ============================================================

scale_ref_630v9r3 = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T
    .copy()
)

sensor_scale_rows_630v9r3 = scale_ref_630v9r3.loc[
    scale_ref_630v9r3["sensor"].astype(str)
    == smoke_sensor_630v9r3
].copy()

if len(sensor_scale_rows_630v9r3) != 1:
    raise RuntimeError(
        "Smoke sensor does not resolve to exactly one corruption-scale row."
    )

scale_row_630v9r3 = sensor_scale_rows_630v9r3.iloc[0]

if not bool(scale_row_630v9r3["usable_gt_1e_12"]):
    raise RuntimeError(
        "Frozen smoke scenario unexpectedly uses a structurally "
        "inadmissible magnitude scale."
    )

smoke_scale_630v9r3 = float(
    scale_row_630v9r3["pooled_within_regime_std"]
)

if (
    not np.isfinite(smoke_scale_630v9r3)
    or smoke_scale_630v9r3 <= 1e-12
):
    raise RuntimeError(
        "Smoke magnitude scale is not finite and > 1e-12."
    )

# Historical bias semantics:
#     clean + direction * alpha * pooled_within_regime_std

smoke_bias_amount_rebuilt_630v9r3 = float(
    smoke_direction_630v9r3
    * smoke_alpha_630v9r3
    * smoke_scale_630v9r3
)

smoke_bias_amount_frozen_630v9r3 = float(
    FD004_SMOKE_BIAS_AMOUNT_630V2
)

smoke_bias_amount_exact_630v9r3 = bool(
    smoke_bias_amount_rebuilt_630v9r3
    == smoke_bias_amount_frozen_630v9r3
)

if not smoke_bias_amount_exact_630v9r3:
    raise RuntimeError(
        "Exact historical bias amount did not reproduce the frozen "
        "smoke bias amount."
    )

# ============================================================
# 4) REBUILD ONLY THE ALREADY-ESTABLISHED SMOKE CORRUPTION
# ============================================================

smoke_raw_rebuilt_630v9r3 = (
    FD004_DEV_RAW_630C
    .copy()
)

if smoke_sensor_630v9r3 not in smoke_raw_rebuilt_630v9r3.columns:
    raise RuntimeError(
        f"Smoke sensor {smoke_sensor_630v9r3!r} missing from FD004 DEV raw."
    )

smoke_raw_rebuilt_630v9r3[smoke_sensor_630v9r3] = (
    smoke_raw_rebuilt_630v9r3[smoke_sensor_630v9r3].to_numpy(
        dtype=np.float64
    )
    + smoke_bias_amount_rebuilt_630v9r3
)

smoke_raw_frozen_630v9r3 = FD004_SMOKE_CORRUPTED_RAW_630V2

if tuple(smoke_raw_rebuilt_630v9r3.columns) != tuple(
    smoke_raw_frozen_630v9r3.columns
):
    raise RuntimeError(
        "Rebuilt/frozen smoke raw columns differ."
    )

smoke_raw_exact_630v9r3 = bool(
    np.array_equal(
        smoke_raw_rebuilt_630v9r3.to_numpy(),
        smoke_raw_frozen_630v9r3.to_numpy(),
        equal_nan=True,
    )
)

if not smoke_raw_exact_630v9r3:
    raw_diff_630v9r3 = np.asarray(
        smoke_raw_rebuilt_630v9r3.to_numpy(dtype=np.float64)
        - smoke_raw_frozen_630v9r3.to_numpy(dtype=np.float64)
    )
    raise RuntimeError(
        "Rebuilt historical bias corruption does not exactly reproduce "
        "the frozen smoke raw matrix. "
        f"Max absolute difference={float(np.nanmax(np.abs(raw_diff_630v9r3)))}"
    )

# ============================================================
# 5) REPRODUCE FROZEN NORMALIZATION BRIDGE
# ============================================================

dev_regime_ids_630v9r3 = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

smoke_norm_rebuilt_630v9r3 = normalize_fd004_raw_630v2(
    smoke_raw_rebuilt_630v9r3.copy(),
    dev_regime_ids_630v9r3,
)

if not isinstance(smoke_norm_rebuilt_630v9r3, pd.DataFrame):
    raise RuntimeError(
        "Normalization bridge did not return a DataFrame."
    )

smoke_norm_frozen_630v9r3 = FD004_SMOKE_NORMALIZED_630V2

if tuple(smoke_norm_rebuilt_630v9r3.columns) != tuple(
    smoke_norm_frozen_630v9r3.columns
):
    raise RuntimeError(
        "Rebuilt/frozen smoke normalized columns differ."
    )

smoke_normalized_exact_630v9r3 = bool(
    np.array_equal(
        smoke_norm_rebuilt_630v9r3.to_numpy(),
        smoke_norm_frozen_630v9r3.to_numpy(),
        equal_nan=True,
    )
)

if not smoke_normalized_exact_630v9r3:
    norm_diff_630v9r3 = np.asarray(
        smoke_norm_rebuilt_630v9r3.to_numpy(dtype=np.float64)
        - smoke_norm_frozen_630v9r3.to_numpy(dtype=np.float64)
    )
    raise RuntimeError(
        "Frozen normalization bridge was not reproduced exactly. "
        f"Max absolute difference={float(np.nanmax(np.abs(norm_diff_630v9r3)))}"
    )

# ============================================================
# 6) REPRODUCE FROZEN WINDOW BRIDGE
# ============================================================

smoke_window_output_630v9r3 = build_windows_fd004_630v2(
    smoke_norm_rebuilt_630v9r3
)

if not isinstance(smoke_window_output_630v9r3, np.ndarray):
    raise RuntimeError(
        "Window bridge returned unexpected type: "
        f"{type(smoke_window_output_630v9r3).__name__}"
    )

smoke_X_rebuilt_630v9r3 = np.asarray(
    smoke_window_output_630v9r3,
    dtype=np.float32,
)

smoke_X_frozen_630v9r3 = np.asarray(
    FD004_SMOKE_X_DEV_630V2,
    dtype=np.float32,
)

if smoke_X_rebuilt_630v9r3.shape != smoke_X_frozen_630v9r3.shape:
    raise RuntimeError(
        "Rebuilt/frozen smoke window shapes differ: "
        f"{smoke_X_rebuilt_630v9r3.shape} vs "
        f"{smoke_X_frozen_630v9r3.shape}"
    )

smoke_X_exact_630v9r3 = bool(
    np.array_equal(
        smoke_X_rebuilt_630v9r3,
        smoke_X_frozen_630v9r3,
        equal_nan=True,
    )
)

if not smoke_X_exact_630v9r3:
    x_diff_630v9r3 = (
        smoke_X_rebuilt_630v9r3.astype(np.float64)
        - smoke_X_frozen_630v9r3.astype(np.float64)
    )
    raise RuntimeError(
        "Frozen smoke window tensor was not reproduced exactly. "
        f"Max absolute difference={float(np.nanmax(np.abs(x_diff_630v9r3)))}"
    )

# ============================================================
# 7) FREEZE REPRODUCTION EVIDENCE
# ============================================================

FD004_SMOKE_REBUILT_BIAS_AMOUNT_630V9R3 = (
    smoke_bias_amount_rebuilt_630v9r3
)

FD004_SMOKE_RAW_REBUILT_630V9R3 = (
    smoke_raw_rebuilt_630v9r3
)

FD004_SMOKE_NORMALIZED_REBUILT_630V9R3 = (
    smoke_norm_rebuilt_630v9r3
)

FD004_SMOKE_X_REBUILT_630V9R3 = (
    smoke_X_rebuilt_630v9r3
)

FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_630V9R3 = {
    "dataset": "FD004",
    "stage": "6.30V-9R3",
    "historical_operator_source_sha256":
        FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2,
    "historical_dispatcher_source_sha256":
        FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2,
    "full_corruption_execution_contract_sha256":
        FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6,
    "scenario_id": smoke_id_630v9r3,
    "scenario_number": smoke_number_630v9r3,
    "family": "bias",
    "sensor": smoke_sensor_630v9r3,
    "alpha": smoke_alpha_630v9r3,
    "direction": smoke_direction_630v9r3,
    "pooled_within_regime_std": smoke_scale_630v9r3,
    "bias_amount": smoke_bias_amount_rebuilt_630v9r3,
    "bias_amount_exact": smoke_bias_amount_exact_630v9r3,
    "raw_exact": smoke_raw_exact_630v9r3,
    "normalized_exact": smoke_normalized_exact_630v9r3,
    "window_exact": smoke_X_exact_630v9r3,
    "frozen_smoke_x_sha256": FD004_SMOKE_X_SHA256_630V2,
    "frozen_smoke_prediction_sha256":
        FD004_SMOKE_PREDICTION_SHA256_630V2,
    "new_model_inference_performed": False,
    "new_mc_execution_performed": False,
    "dev_targets_errors_used": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_SHA256_630V9R3 = (
    hashlib.sha256(
        json.dumps(
            FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_630V9R3,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_SMOKE_BRIDGE_EXACTLY_REPRODUCED_630V9R3 = True
FD004_FULL_CORRUPTION_EXECUTED_630V9R3 = False
FD004_NEW_MODEL_INFERENCE_PERFORMED_630V9R3 = False
FD004_NEW_MC_EXECUTION_PERFORMED_630V9R3 = False
FD004_DEV_TARGETS_ERRORS_USED_630V9R3 = False
FD004_THRESHOLD_SELECTED_630V9R3 = False
FD004_POLICY_SELECTED_630V9R3 = False
FD004_TEST_CONTENTS_READ_630V9R3 = False
FD004_RUL_CONTENTS_READ_630V9R3 = False

all_checks_passed_630v9r3 = True

# ============================================================
# 8) REPORT
# ============================================================

print("=" * 176)
print(
    "STEP 6.30V-9R3 — FD004 EXACT HISTORICAL BIAS-SMOKE "
    "RAW→NORMALIZATION→WINDOW REPRODUCTION LOCK"
)
print("=" * 176)

print("Full corruption run executed     :", False)
print("New model inference performed    :", False)
print("New MC execution performed       :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- FROZEN SMOKE SCENARIO ---")
print(
    "Scenario ID / number             :",
    (smoke_id_630v9r3, smoke_number_630v9r3),
)
print(
    "Family / sensor                  :",
    ("bias", smoke_sensor_630v9r3),
)
print(
    "Alpha / direction                :",
    (smoke_alpha_630v9r3, smoke_direction_630v9r3),
)
print(
    "Pooled within-regime scale       :",
    smoke_scale_630v9r3,
)
print(
    "Rebuilt / frozen bias amount     :",
    (
        smoke_bias_amount_rebuilt_630v9r3,
        smoke_bias_amount_frozen_630v9r3,
    ),
)

print("\n--- EXACT BRIDGE REPRODUCTION ---")
print("Bias amount exact                :", smoke_bias_amount_exact_630v9r3)
print("Corrupted raw exact              :", smoke_raw_exact_630v9r3)
print("Normalized matrix exact          :", smoke_normalized_exact_630v9r3)
print("Window tensor exact              :", smoke_X_exact_630v9r3)
print(
    "Window shape                     :",
    smoke_X_rebuilt_630v9r3.shape,
)

print("\n--- FROZEN DOWNSTREAM AUTHORITY ---")
print(
    "Frozen smoke X SHA-256           :",
    FD004_SMOKE_X_SHA256_630V2,
)
print(
    "Frozen smoke prediction SHA-256  :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)

print("\n--- CONTRACT FINGERPRINT ---")
print(
    "Smoke reproduction SHA-256       :",
    FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_SHA256_630V9R3,
)

print(
    "\nPASS — the exact historical bias semantics reproduce the "
    "already-established FD004 smoke corruption, frozen normalization, "
    "and frozen window tensor exactly."
)

print(
    "No new model inference or MC was needed; the previously frozen "
    "smoke prediction remains the downstream authority."
)

print(
    "Next: reconstruct and lock the remaining noise/drift/dropout/stuck "
    "operators from their exact historical source semantics, with "
    "structural self-checks, before any 381-scenario execution."
)

In [ ]:
# STEP 6.30V-9R4
# FD004 WINDOW-BRIDGE RETURN-STRUCTURE AUDIT
# FROM THE ALREADY-RETURNED SMOKE OBJECT
#
# No helper rerun.
# No corruption rerun.
# No normalization rerun.
# No inference / MC / DEV scoring / threshold / TEST / RUL.

import numpy as np

required_630v9r4 = (
    "smoke_window_output_630v9r3",
    "FD004_SMOKE_X_DEV_630V2",
    "FD004_SMOKE_X_SHA256_630V2",
)

missing_630v9r4 = [
    name_630v9r4
    for name_630v9r4 in required_630v9r4
    if name_630v9r4 not in globals()
]

if missing_630v9r4:
    raise RuntimeError(
        "Missing STEP 6.30V-9R4 authority object(s): "
        + ", ".join(missing_630v9r4)
    )

returned_630v9r4 = smoke_window_output_630v9r3

if not isinstance(returned_630v9r4, tuple):
    raise RuntimeError(
        "Expected the already-returned window bridge object "
        f"to be a tuple; observed {type(returned_630v9r4).__name__}."
    )

frozen_x_630v9r4 = np.asarray(
    FD004_SMOKE_X_DEV_630V2,
    dtype=np.float32,
)

candidate_x_indices_630v9r4 = []
rows_630v9r4 = []

for idx_630v9r4, item_630v9r4 in enumerate(returned_630v9r4):

    item_type_630v9r4 = type(item_630v9r4).__name__
    item_shape_630v9r4 = getattr(item_630v9r4, "shape", None)
    item_dtype_630v9r4 = getattr(item_630v9r4, "dtype", None)

    exact_frozen_x_match_630v9r4 = False

    if isinstance(item_630v9r4, np.ndarray):

        if tuple(item_630v9r4.shape) == tuple(frozen_x_630v9r4.shape):

            candidate_630v9r4 = np.asarray(
                item_630v9r4,
                dtype=np.float32,
            )

            exact_frozen_x_match_630v9r4 = bool(
                np.array_equal(
                    candidate_630v9r4,
                    frozen_x_630v9r4,
                    equal_nan=True,
                )
            )

            if exact_frozen_x_match_630v9r4:
                candidate_x_indices_630v9r4.append(
                    idx_630v9r4
                )

    rows_630v9r4.append(
        {
            "index": idx_630v9r4,
            "type": item_type_630v9r4,
            "shape": (
                None
                if item_shape_630v9r4 is None
                else tuple(item_shape_630v9r4)
            ),
            "dtype": (
                None
                if item_dtype_630v9r4 is None
                else str(item_dtype_630v9r4)
            ),
            "exact_frozen_X_match":
                exact_frozen_x_match_630v9r4,
        }
    )

if len(candidate_x_indices_630v9r4) != 1:
    raise RuntimeError(
        "Could not identify exactly one tuple element matching "
        "the frozen smoke X tensor. Matching indices="
        + str(candidate_x_indices_630v9r4)
    )

FD004_WINDOW_BRIDGE_X_RETURN_INDEX_630V9R4 = int(
    candidate_x_indices_630v9r4[0]
)

FD004_WINDOW_BRIDGE_RETURN_STRUCTURE_630V9R4 = tuple(
    (
        row_630v9r4["index"],
        row_630v9r4["type"],
        row_630v9r4["shape"],
        row_630v9r4["dtype"],
    )
    for row_630v9r4 in rows_630v9r4
)

FD004_WINDOW_BRIDGE_X_RETURN_INDEX_LOCKED_630V9R4 = True

FD004_CORRUPTION_RERUN_630V9R4 = False
FD004_NORMALIZATION_RERUN_630V9R4 = False
FD004_WINDOW_HELPER_RERUN_630V9R4 = False
FD004_MODEL_INFERENCE_PERFORMED_630V9R4 = False
FD004_MC_EXECUTION_PERFORMED_630V9R4 = False
FD004_DEV_TARGETS_ERRORS_USED_630V9R4 = False
FD004_THRESHOLD_SELECTED_630V9R4 = False
FD004_POLICY_SELECTED_630V9R4 = False
FD004_TEST_CONTENTS_READ_630V9R4 = False
FD004_RUL_CONTENTS_READ_630V9R4 = False

all_checks_passed_630v9r4 = True

print("=" * 176)
print(
    "STEP 6.30V-9R4 — FD004 WINDOW-BRIDGE RETURN-STRUCTURE AUDIT "
    "FROM ALREADY-RETURNED SMOKE OBJECT"
)
print("=" * 176)

print("Corruption rerun                 :", False)
print("Normalization rerun              :", False)
print("Window helper rerun              :", False)
print("Model inference performed        :", False)
print("MC execution performed           :", False)
print("DEV targets/errors used          :", False)
print("Threshold selected               :", False)
print("Policy selected                  :", False)
print("TEST contents read               :", False)
print("Official RUL contents read       :", False)

print("\n--- RETURN STRUCTURE ---")
print(
    "Python type                      :",
    type(returned_630v9r4).__name__,
)
print(
    "Tuple item count                 :",
    len(returned_630v9r4),
)

for row_630v9r4 in rows_630v9r4:
    print(
        f"item {row_630v9r4['index']}: "
        f"type={row_630v9r4['type']}, "
        f"shape={row_630v9r4['shape']}, "
        f"dtype={row_630v9r4['dtype']}, "
        f"exact_frozen_X_match="
        f"{row_630v9r4['exact_frozen_X_match']}"
    )

print("\n--- RESOLVED WINDOW AUTHORITY ---")
print(
    "Frozen X tensor return index     :",
    FD004_WINDOW_BRIDGE_X_RETURN_INDEX_630V9R4,
)
print(
    "Frozen X shape                   :",
    frozen_x_630v9r4.shape,
)
print(
    "Frozen X SHA-256 authority       :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "\nPASS — the exact tuple position containing the already-frozen "
    "FD004 smoke window tensor is now resolved without rerunning any "
    "experimental operation."
)

In [ ]:
# ============================================================

# STEP 6.30V-9R5
# FD004 SMOKE-REPRODUCTION RECOVERY + FINAL LOCK
#
# Uses only already-resident outputs from the prior failed 9R3 cell
# plus the return-structure resolution from 9R4.
#
# NO corruption rerun.
# NO normalization rerun.
# NO window-helper rerun.
# NO model inference.
# NO MC.
# NO DEV outcome/error use.
# NO threshold/policy.
# NO TEST / RUL.
# ============================================================

import numpy as np
import hashlib
import json


# ============================================================
# 1. PRECONDITIONS
# ============================================================

required_630v9r5 = (
    "all_checks_passed_630v9r4",
    "all_checks_passed_630v9r2",

    "smoke_bias_amount_rebuilt_630v9r3",
    "smoke_bias_amount_frozen_630v9r3",
    "smoke_bias_amount_exact_630v9r3",
    "smoke_raw_exact_630v9r3",
    "smoke_normalized_exact_630v9r3",
    "smoke_window_output_630v9r3",

    "smoke_id_630v9r3",
    "smoke_number_630v9r3",
    "smoke_sensor_630v9r3",
    "smoke_alpha_630v9r3",
    "smoke_direction_630v9r3",
    "smoke_scale_630v9r3",

    "FD004_WINDOW_BRIDGE_X_RETURN_INDEX_630V9R4",

    "FD004_SMOKE_X_DEV_630V2",
    "FD004_SMOKE_X_SHA256_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",

    "FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2",
    "FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2",

    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",
)

missing_630v9r5 = [
    name_630v9r5
    for name_630v9r5 in required_630v9r5
    if name_630v9r5 not in globals()
]

if missing_630v9r5:
    raise RuntimeError(
        "Missing STEP 6.30V-9R5 authority object(s): "
        + ", ".join(missing_630v9r5)
    )

if not bool(all_checks_passed_630v9r4):
    raise RuntimeError(
        "STEP 6.30V-9R4 must pass first."
    )

if not bool(all_checks_passed_630v9r2):
    raise RuntimeError(
        "STEP 6.30V-9R2 must pass first."
    )


# ============================================================
# 2. VERIFY EARLIER 9R3 CHECKS THAT COMPLETED BEFORE FAILURE
# ============================================================

if not bool(smoke_bias_amount_exact_630v9r3):
    raise RuntimeError(
        "Previously completed exact bias-amount check is False."
    )

if not bool(smoke_raw_exact_630v9r3):
    raise RuntimeError(
        "Previously completed exact raw-corruption check is False."
    )

if not bool(smoke_normalized_exact_630v9r3):
    raise RuntimeError(
        "Previously completed exact normalization check is False."
    )


# ============================================================
# 3. RESOLVE X FROM THE ALREADY-RETURNED WINDOW TUPLE
# ============================================================

x_index_630v9r5 = int(
    FD004_WINDOW_BRIDGE_X_RETURN_INDEX_630V9R4
)

if x_index_630v9r5 != 0:
    raise RuntimeError(
        "Frozen window-X tuple index unexpectedly changed."
    )

window_output_630v9r5 = (
    smoke_window_output_630v9r3
)

if not isinstance(
    window_output_630v9r5,
    tuple,
):
    raise RuntimeError(
        "Resident smoke window output is no longer a tuple."
    )

if len(window_output_630v9r5) != 3:
    raise RuntimeError(
        "Resident smoke window output no longer has 3 items."
    )

smoke_X_recovered_630v9r5 = np.asarray(
    window_output_630v9r5[
        x_index_630v9r5
    ],
    dtype=np.float32,
)

smoke_X_frozen_630v9r5 = np.asarray(
    FD004_SMOKE_X_DEV_630V2,
    dtype=np.float32,
)

if (
    smoke_X_recovered_630v9r5.shape
    !=
    smoke_X_frozen_630v9r5.shape
):
    raise RuntimeError(
        "Recovered/frozen smoke X shapes differ: "
        f"{smoke_X_recovered_630v9r5.shape} vs "
        f"{smoke_X_frozen_630v9r5.shape}"
    )

smoke_X_exact_630v9r5 = bool(
    np.array_equal(
        smoke_X_recovered_630v9r5,
        smoke_X_frozen_630v9r5,
        equal_nan=True,
    )
)

if not smoke_X_exact_630v9r5:
    raise RuntimeError(
        "Recovered window tensor does not exactly match "
        "the frozen FD004 smoke X authority."
    )


# ============================================================
# 4. VERIFY AUXILIARY RETURN STRUCTURE
# ============================================================

smoke_window_unit_ids_630v9r5 = np.asarray(
    window_output_630v9r5[1]
)

smoke_endpoint_cycles_630v9r5 = np.asarray(
    window_output_630v9r5[2]
)

if smoke_window_unit_ids_630v9r5.shape != (10824,):
    raise RuntimeError(
        "Unexpected smoke window-unit-ID shape."
    )

if smoke_endpoint_cycles_630v9r5.shape != (10824,):
    raise RuntimeError(
        "Unexpected smoke endpoint-cycle shape."
    )


# ============================================================
# 5. FREEZE RECOVERED SMOKE AUTHORITY
# ============================================================

FD004_SMOKE_X_REBUILT_630V9R5 = (
    smoke_X_recovered_630v9r5
)

FD004_SMOKE_WINDOW_UNIT_IDS_REBUILT_630V9R5 = (
    smoke_window_unit_ids_630v9r5
)

FD004_SMOKE_ENDPOINT_CYCLES_REBUILT_630V9R5 = (
    smoke_endpoint_cycles_630v9r5
)


FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_630V9R5 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-9R5",

    "historical_operator_source_sha256":
        str(
            FD004_HISTORICAL_OPERATOR_SOURCE_SHA256_630V8R2
        ),

    "historical_dispatcher_source_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_SOURCE_SHA256_630V8R2
        ),

    "full_corruption_execution_contract_sha256":
        str(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6
        ),

    "scenario_id":
        str(
            smoke_id_630v9r3
        ),

    "scenario_number":
        int(
            smoke_number_630v9r3
        ),

    "family":
        "bias",

    "sensor":
        str(
            smoke_sensor_630v9r3
        ),

    "alpha":
        float(
            smoke_alpha_630v9r3
        ),

    "direction":
        int(
            smoke_direction_630v9r3
        ),

    "pooled_within_regime_std":
        float(
            smoke_scale_630v9r3
        ),

    "rebuilt_bias_amount":
        float(
            smoke_bias_amount_rebuilt_630v9r3
        ),

    "frozen_bias_amount":
        float(
            smoke_bias_amount_frozen_630v9r3
        ),

    "bias_amount_exact":
        True,

    "raw_corruption_exact":
        True,

    "normalization_exact":
        True,

    "window_tensor_exact":
        True,

    "window_helper_return_structure":
        "tuple(X, window_unit_ids, endpoint_cycles)",

    "window_X_return_index":
        0,

    "window_X_shape":
        tuple(
            smoke_X_recovered_630v9r5.shape
        ),

    "window_unit_ids_shape":
        tuple(
            smoke_window_unit_ids_630v9r5.shape
        ),

    "endpoint_cycles_shape":
        tuple(
            smoke_endpoint_cycles_630v9r5.shape
        ),

    "frozen_smoke_x_sha256":
        str(
            FD004_SMOKE_X_SHA256_630V2
        ),

    "frozen_smoke_prediction_sha256":
        str(
            FD004_SMOKE_PREDICTION_SHA256_630V2
        ),

    "corruption_rerun":
        False,

    "normalization_rerun":
        False,

    "window_helper_rerun":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_SHA256_630V9R5 = (
    hashlib.sha256(
        json.dumps(
            FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_630V9R5,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


# ============================================================
# 6. FINAL LOCK FLAGS
# ============================================================

FD004_SMOKE_BRIDGE_EXACTLY_REPRODUCED_630V9R5 = True

FD004_HISTORICAL_BIAS_SMOKE_VALIDATED_630V9R5 = True

FD004_CORRUPTION_RERUN_630V9R5 = False
FD004_NORMALIZATION_RERUN_630V9R5 = False
FD004_WINDOW_HELPER_RERUN_630V9R5 = False

FD004_FULL_CORRUPTION_EXECUTED_630V9R5 = False

FD004_NEW_MODEL_INFERENCE_PERFORMED_630V9R5 = False
FD004_NEW_MC_EXECUTION_PERFORMED_630V9R5 = False
FD004_DEV_TARGETS_ERRORS_USED_630V9R5 = False

FD004_THRESHOLD_SELECTED_630V9R5 = False
FD004_POLICY_SELECTED_630V9R5 = False

FD004_TEST_CONTENTS_READ_630V9R5 = False
FD004_RUL_CONTENTS_READ_630V9R5 = False

all_checks_passed_630v9r5 = True


# ============================================================
# 7. REPORT
# ============================================================

print("=" * 176)

print(
    "STEP 6.30V-9R5 — FD004 SMOKE-REPRODUCTION "
    "RECOVERY + FINAL LOCK"
)

print("=" * 176)

print(
    "Corruption rerun                  :",
    False,
)

print(
    "Normalization rerun               :",
    False,
)

print(
    "Window helper rerun               :",
    False,
)

print(
    "New model inference performed     :",
    False,
)

print(
    "New MC execution performed        :",
    False,
)

print(
    "DEV targets/errors used           :",
    False,
)

print(
    "Threshold selected                :",
    False,
)

print(
    "Policy selected                   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print(
    "\n--- EXISTING 9R3 CHECKS ---"
)

print(
    "Bias amount exact                 :",
    smoke_bias_amount_exact_630v9r3,
)

print(
    "Raw corruption exact              :",
    smoke_raw_exact_630v9r3,
)

print(
    "Normalization exact               :",
    smoke_normalized_exact_630v9r3,
)


print(
    "\n--- RESOLVED WINDOW BRIDGE ---"
)

print(
    "Return structure                  :",
    "tuple(X, window_unit_ids, endpoint_cycles)",
)

print(
    "X return index                    :",
    x_index_630v9r5,
)

print(
    "X shape                           :",
    smoke_X_recovered_630v9r5.shape,
)

print(
    "Window-unit IDs shape             :",
    smoke_window_unit_ids_630v9r5.shape,
)

print(
    "Endpoint cycles shape             :",
    smoke_endpoint_cycles_630v9r5.shape,
)

print(
    "Frozen X exact                    :",
    smoke_X_exact_630v9r5,
)


print(
    "\n--- SMOKE AUTHORITY ---"
)

print(
    "Scenario ID / number              :",
    (
        smoke_id_630v9r3,
        smoke_number_630v9r3,
    ),
)

print(
    "Family / sensor                   :",
    (
        "bias",
        smoke_sensor_630v9r3,
    ),
)

print(
    "Rebuilt / frozen bias amount      :",
    (
        smoke_bias_amount_rebuilt_630v9r3,
        smoke_bias_amount_frozen_630v9r3,
    ),
)

print(
    "Frozen smoke X SHA-256            :",
    FD004_SMOKE_X_SHA256_630V2,
)

print(
    "Frozen smoke prediction SHA-256   :",
    FD004_SMOKE_PREDICTION_SHA256_630V2,
)


print(
    "\n--- CONTRACT FINGERPRINT ---"
)

print(
    "Smoke reproduction SHA-256        :",
    FD004_SMOKE_BRIDGE_REPRODUCTION_CONTRACT_SHA256_630V9R5,
)


print(
    "\nPASS — the historical FD004 bias smoke scenario is now "
    "fully and exactly reproduced through raw corruption, frozen "
    "normalization, and frozen window construction."
)

print(
    "The recovery reused already-resident intermediate results; "
    "no corruption, normalization, or window helper was rerun."
)

print(
    "Next: reconstruct the remaining noise, drift, dropout, and "
    "stuck operators from the exact recovered historical source "
    "semantics and validate their structural invariants before "
    "starting the 381-scenario execution."
)

In [ ]:
# STEP 6.30V-10
# FD004 EXACT HISTORICAL FIVE-FAMILY OPERATOR SOURCE EXTRACTION + LOCK
#
# SOURCE INSPECTION ONLY.
# No corruption.
# No operator execution.
# No preprocessing.
# No model inference.
# No MC.
# No DEV outcomes/errors.
# No threshold/policy.
# No TEST/RUL.

import ast
import hashlib

required_630v10 = (
    "all_checks_passed_630v9r5",
    "FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2",
    "cells_630tr2",
)

missing_630v10 = [
    name_630v10
    for name_630v10 in required_630v10
    if name_630v10 not in globals()
]

if missing_630v10:
    raise RuntimeError(
        "Missing STEP 6.30V-10 authority object(s): "
        + ", ".join(missing_630v10)
    )

if not bool(all_checks_passed_630v9r5):
    raise RuntimeError(
        "STEP 6.30V-9R5 must pass before STEP 6.30V-10."
    )

authority_hits_630v10 = (
    FD004_HISTORICAL_CORRUPTION_OPERATOR_SOURCE_HITS_630TR2
)

if not isinstance(authority_hits_630v10, tuple):
    raise RuntimeError(
        "Historical operator-source authority is not a tuple."
    )

operator_hits_630v10 = [
    hit_630v10
    for hit_630v10 in authority_hits_630v10
    if isinstance(hit_630v10, dict)
    and tuple(hit_630v10.get("defined_functions", ())) == (
        "apply_fd002_bias_628gc",
        "apply_fd002_noise_628gc",
        "apply_fd002_drift_628gc",
        "apply_fd002_dropout_628gc",
        "apply_fd002_stuck_628gc",
    )
]

if len(operator_hits_630v10) != 1:
    raise RuntimeError(
        "Could not resolve exactly one historical five-family "
        "operator-source authority."
    )

operator_hit_630v10 = operator_hits_630v10[0]

operator_cell_index_630v10 = int(
    operator_hit_630v10["cell_index"]
)

operator_source_sha_authority_630v10 = str(
    operator_hit_630v10["source_sha256"]
)

EXPECTED_OPERATOR_SOURCE_SHA256_630V10 = (
    "74dc1af52e2359a74191cb3121915b89"
    "bb42f4522e1add88e39b1c82e68ced2e"
)

if (
    operator_source_sha_authority_630v10
    != EXPECTED_OPERATOR_SOURCE_SHA256_630V10
):
    raise RuntimeError(
        "Recovered historical operator authority SHA-256 "
        "does not match the previously locked value."
    )

if not (
    0
    <= operator_cell_index_630v10
    < len(cells_630tr2)
):
    raise RuntimeError(
        "Historical operator cell index is outside the "
        "recovered notebook-cell range."
    )

operator_cell_630v10 = cells_630tr2[
    operator_cell_index_630v10
]

if not isinstance(operator_cell_630v10, dict):
    raise RuntimeError(
        "Recovered operator cell is not a notebook-cell dict."
    )

source_field_630v10 = operator_cell_630v10.get(
    "source",
    None,
)

if source_field_630v10 is None:
    raise RuntimeError(
        "Recovered operator cell has no source field."
    )

if isinstance(source_field_630v10, list):
    operator_source_630v10 = "".join(
        str(part_630v10)
        for part_630v10 in source_field_630v10
    )
elif isinstance(source_field_630v10, str):
    operator_source_630v10 = source_field_630v10
else:
    raise RuntimeError(
        "Recovered operator-cell source has unexpected type: "
        f"{type(source_field_630v10).__name__}"
    )

operator_source_sha_observed_630v10 = hashlib.sha256(
    operator_source_630v10.encode("utf-8")
).hexdigest()

if (
    operator_source_sha_observed_630v10
    != operator_source_sha_authority_630v10
):
    raise RuntimeError(
        "Recovered historical operator-cell source does not "
        "match its locked SHA-256 authority."
    )

try:
    operator_tree_630v10 = ast.parse(
        operator_source_630v10
    )
except SyntaxError as exc_630v10:
    raise RuntimeError(
        "Locked historical operator source cannot be parsed."
    ) from exc_630v10

target_functions_630v10 = (
    "apply_fd002_bias_628gc",
    "apply_fd002_noise_628gc",
    "apply_fd002_drift_628gc",
    "apply_fd002_dropout_628gc",
    "apply_fd002_stuck_628gc",
)

function_nodes_630v10 = {
    node_630v10.name: node_630v10
    for node_630v10 in operator_tree_630v10.body
    if isinstance(
        node_630v10,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    )
    and node_630v10.name in target_functions_630v10
}

missing_functions_630v10 = [
    name_630v10
    for name_630v10 in target_functions_630v10
    if name_630v10 not in function_nodes_630v10
]

if missing_functions_630v10:
    raise RuntimeError(
        "Historical operator source is missing target function(s): "
        + ", ".join(missing_functions_630v10)
    )

extracted_sources_630v10 = {}
extracted_hashes_630v10 = {}
signatures_ast_630v10 = {}

source_lines_630v10 = operator_source_630v10.splitlines(
    keepends=True
)

for function_name_630v10 in target_functions_630v10:
    node_630v10 = function_nodes_630v10[
        function_name_630v10
    ]

    if (
        getattr(node_630v10, "lineno", None) is None
        or getattr(node_630v10, "end_lineno", None) is None
    ):
        raise RuntimeError(
            f"AST source boundaries unavailable for "
            f"{function_name_630v10}."
        )

    start_630v10 = int(node_630v10.lineno) - 1
    end_630v10 = int(node_630v10.end_lineno)

    fn_source_630v10 = "".join(
        source_lines_630v10[
            start_630v10:end_630v10
        ]
    )

    extracted_sources_630v10[
        function_name_630v10
    ] = fn_source_630v10

    extracted_hashes_630v10[
        function_name_630v10
    ] = hashlib.sha256(
        fn_source_630v10.encode("utf-8")
    ).hexdigest()

    args_630v10 = [
        arg_630v10.arg
        for arg_630v10 in node_630v10.args.args
    ]

    defaults_count_630v10 = len(
        node_630v10.args.defaults
    )

    signatures_ast_630v10[
        function_name_630v10
    ] = {
        "positional_args": tuple(args_630v10),
        "default_count": int(defaults_count_630v10),
        "has_vararg": (
            node_630v10.args.vararg is not None
        ),
        "has_kwarg": (
            node_630v10.args.kwarg is not None
        ),
    }

FD004_HISTORICAL_OPERATOR_CELL_INDEX_630V10 = (
    operator_cell_index_630v10
)

FD004_HISTORICAL_OPERATOR_CELL_SOURCE_SHA256_630V10 = (
    operator_source_sha_observed_630v10
)

FD004_HISTORICAL_OPERATOR_EXACT_SOURCES_630V10 = {
    key_630v10: value_630v10
    for key_630v10, value_630v10
    in extracted_sources_630v10.items()
}

FD004_HISTORICAL_OPERATOR_EXACT_SOURCE_SHA256_630V10 = {
    key_630v10: value_630v10
    for key_630v10, value_630v10
    in extracted_hashes_630v10.items()
}

FD004_HISTORICAL_OPERATOR_AST_SIGNATURES_630V10 = {
    key_630v10: dict(value_630v10)
    for key_630v10, value_630v10
    in signatures_ast_630v10.items()
}

FD004_HISTORICAL_OPERATOR_SOURCE_EXTRACTION_LOCKED_630V10 = True

FD004_CORRUPTION_EXECUTED_630V10 = False
FD004_OPERATOR_EXECUTED_630V10 = False
FD004_MODEL_INFERENCE_PERFORMED_630V10 = False
FD004_MC_EXECUTION_PERFORMED_630V10 = False
FD004_DEV_TARGETS_ERRORS_USED_630V10 = False
FD004_THRESHOLD_SELECTED_630V10 = False
FD004_POLICY_SELECTED_630V10 = False
FD004_TEST_CONTENTS_READ_630V10 = False
FD004_RUL_CONTENTS_READ_630V10 = False

all_checks_passed_630v10 = True

print("=" * 176)
print(
    "STEP 6.30V-10 — FD004 EXACT HISTORICAL FIVE-FAMILY "
    "OPERATOR SOURCE EXTRACTION + LOCK"
)
print("=" * 176)

print("Corruption executed               :", False)
print("Historical operator executed      :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- SOURCE AUTHORITY ---")
print(
    "Historical cell index             :",
    FD004_HISTORICAL_OPERATOR_CELL_INDEX_630V10,
)
print(
    "Cell-source SHA-256               :",
    FD004_HISTORICAL_OPERATOR_CELL_SOURCE_SHA256_630V10,
)
print(
    "Source fingerprint exact          :",
    (
        FD004_HISTORICAL_OPERATOR_CELL_SOURCE_SHA256_630V10
        == EXPECTED_OPERATOR_SOURCE_SHA256_630V10
    ),
)

print("\n--- EXACT HISTORICAL FUNCTION DEFINITIONS ---")

for rank_630v10, function_name_630v10 in enumerate(
    target_functions_630v10,
    start=1,
):
    print("\n" + "-" * 120)
    print(
        f"[{rank_630v10}] {function_name_630v10}"
    )
    print(
        "AST signature metadata:",
        FD004_HISTORICAL_OPERATOR_AST_SIGNATURES_630V10[
            function_name_630v10
        ],
    )
    print(
        "Function-source SHA-256:",
        FD004_HISTORICAL_OPERATOR_EXACT_SOURCE_SHA256_630V10[
            function_name_630v10
        ],
    )
    print("-" * 120)
    print(
        FD004_HISTORICAL_OPERATOR_EXACT_SOURCES_630V10[
            function_name_630v10
        ].rstrip()
    )

print(
    "\nPASS — the exact historical bias/noise/drift/dropout/stuck "
    "function definitions were extracted from the locked historical "
    "source cell and fingerprinted without executing them."
)

print(
    "Next: use these exact definitions—not inferred semantics—to "
    "construct the FD004 operator implementation and validate each "
    "family structurally before any full corruption block is executed."
)

In [ ]:
# ==================================================================================================
# STEP 6.30V-11
# FD004 EXACT-SEMANTICS FIVE-FAMILY CORRUPTION-OPERATOR IMPLEMENTATION LOCK
#
# Defines operators only.
# NO corruption scenario is executed.
# NO model inference.
# NO MC.
# NO DEV outcome/error use.
# NO threshold/policy selection.
# NO TEST / official RUL access.
# ==================================================================================================

import numpy as np
import pandas as pd
import hashlib
import json
import inspect
from types import MappingProxyType


# ==================================================================================================
# 0) PRECONDITIONS
# ==================================================================================================

required_630v11 = (
    "all_checks_passed_630v10",
    "all_checks_passed_630v9r5",

    "FD004_CORRUPTION_SCALE_REFERENCE_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_630T",
    "FD004_SENSOR_COLS_630E",

    "FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T",
    "FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T",
)

missing_630v11 = [
    name_630v11
    for name_630v11 in required_630v11
    if name_630v11 not in globals()
]

if missing_630v11:
    raise RuntimeError(
        "Missing STEP 6.30V-11 authority object(s): "
        + ", ".join(missing_630v11)
    )

if not bool(all_checks_passed_630v10):
    raise RuntimeError(
        "STEP 6.30V-10 must pass before STEP 6.30V-11."
    )

if not bool(all_checks_passed_630v9r5):
    raise RuntimeError(
        "STEP 6.30V-9R5 must remain passed."
    )


# ==================================================================================================
# 1) FREEZE FD004 SENSOR AUTHORITY
# ==================================================================================================

sensor_cols_630v11 = tuple(
    str(sensor_630v11)
    for sensor_630v11 in FD004_SENSOR_COLS_630E
)

if (
    len(sensor_cols_630v11) != 21
    or len(set(sensor_cols_630v11)) != 21
):
    raise RuntimeError(
        "Frozen FD004 physical-sensor authority is not "
        "exactly 21 unique sensors."
    )


# ==================================================================================================
# 2) BUILD FD004 FIT-ONLY SENSOR-SCALE LOOKUP
# ==================================================================================================

scale_ref_630v11 = (
    FD004_CORRUPTION_SCALE_REFERENCE_630T
)

if not isinstance(
    scale_ref_630v11,
    pd.DataFrame,
):
    raise RuntimeError(
        "FD004 corruption-scale reference is not a DataFrame."
    )

required_scale_cols_630v11 = (
    "sensor",
    "pooled_within_regime_std",
)

if any(
    col_630v11 not in scale_ref_630v11.columns
    for col_630v11 in required_scale_cols_630v11
):
    raise RuntimeError(
        "FD004 corruption-scale reference lacks required columns."
    )

scale_map_work_630v11 = {}

for row_630v11 in (
    scale_ref_630v11
    .loc[:, required_scale_cols_630v11]
    .itertuples(index=False)
):
    sensor_630v11 = str(
        row_630v11.sensor
    )

    if sensor_630v11 in scale_map_work_630v11:
        raise RuntimeError(
            f"Duplicate scale-reference sensor: "
            f"{sensor_630v11}"
        )

    value_630v11 = float(
        row_630v11.pooled_within_regime_std
    )

    if (
        not np.isfinite(value_630v11)
        or value_630v11 < 0.0
    ):
        raise RuntimeError(
            f"Invalid pooled scale for "
            f"{sensor_630v11}: {value_630v11}"
        )

    scale_map_work_630v11[
        sensor_630v11
    ] = value_630v11


if (
    set(scale_map_work_630v11)
    != set(sensor_cols_630v11)
):
    raise RuntimeError(
        "FD004 scale-reference sensor set does not "
        "exactly match frozen sensor authority."
    )


# ==================================================================================================
# 3) BUILD FD004 FIT-ONLY DROPOUT IMPUTATION LOOKUP
# ==================================================================================================

drop_ref_630v11 = (
    FD004_DROPOUT_IMPUTATION_REFERENCE_630T
)

if not isinstance(
    drop_ref_630v11,
    pd.DataFrame,
):
    raise RuntimeError(
        "FD004 dropout-imputation reference is not a DataFrame."
    )

required_drop_cols_630v11 = (
    "regime_id",
    "sensor",
    "mean",
)

if any(
    col_630v11 not in drop_ref_630v11.columns
    for col_630v11 in required_drop_cols_630v11
):
    raise RuntimeError(
        "FD004 dropout-imputation reference lacks required columns."
    )

drop_map_work_630v11 = {}

for row_630v11 in (
    drop_ref_630v11
    .loc[:, required_drop_cols_630v11]
    .itertuples(index=False)
):
    key_630v11 = (
        int(row_630v11.regime_id),
        str(row_630v11.sensor),
    )

    if key_630v11 in drop_map_work_630v11:
        raise RuntimeError(
            f"Duplicate dropout-imputation key: "
            f"{key_630v11}"
        )

    mean_630v11 = float(
        row_630v11.mean
    )

    if not np.isfinite(mean_630v11):
        raise RuntimeError(
            f"Non-finite dropout-imputation mean "
            f"for {key_630v11}."
        )

    drop_map_work_630v11[
        key_630v11
    ] = mean_630v11


expected_drop_keys_630v11 = {
    (
        regime_id_630v11,
        sensor_630v11,
    )
    for regime_id_630v11 in range(6)
    for sensor_630v11 in sensor_cols_630v11
}

if (
    set(drop_map_work_630v11)
    != expected_drop_keys_630v11
):
    missing_keys_630v11 = sorted(
        expected_drop_keys_630v11
        - set(drop_map_work_630v11)
    )

    extra_keys_630v11 = sorted(
        set(drop_map_work_630v11)
        - expected_drop_keys_630v11
    )

    raise RuntimeError(
        "FD004 dropout-imputation key grid is not exact. "
        f"missing={missing_keys_630v11[:5]}, "
        f"extra={extra_keys_630v11[:5]}"
    )


FD004_SENSOR_COLS_630V11 = (
    sensor_cols_630v11
)

FD004_SENSOR_SCALE_630V11 = (
    MappingProxyType(
        dict(scale_map_work_630v11)
    )
)

FD004_DROPOUT_MEAN_630V11 = (
    MappingProxyType(
        dict(drop_map_work_630v11)
    )
)


# ==================================================================================================
# 4) EXACT-SEMANTICS FD004 OPERATOR SOURCE
# ==================================================================================================

FD004_FIVE_OPERATOR_SOURCE_630V11 = r"""
def validate_sensor_fd004_630v11(df, sensor):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            "df must be a pandas DataFrame."
        )

    if sensor not in FD004_SENSOR_COLS_630V11:
        raise ValueError(
            f"Unknown/nonphysical FD004 sensor: {sensor}."
        )

    if sensor not in df.columns:
        raise ValueError(
            f"Sensor column missing from dataframe: {sensor}."
        )


def apply_fd004_bias_630v11(
    df,
    sensor,
    alpha,
    direction
):
    validate_sensor_fd004_630v11(
        df,
        sensor
    )

    if direction not in (
        -1,
        1
    ):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(
        alpha
    )

    if (
        not np.isfinite(alpha)
        or alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    scale = FD004_SENSOR_SCALE_630V11[
        sensor
    ]

    bias_amount = float(
        direction
        * alpha
        * scale
    )

    corrupted = df.copy()

    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        bias_amount
    )

    return (
        corrupted,
        bias_amount
    )


def apply_fd004_noise_630v11(
    df,
    sensor,
    alpha,
    seed=42
):
    validate_sensor_fd004_630v11(
        df,
        sensor
    )

    alpha = float(
        alpha
    )

    if (
        not np.isfinite(alpha)
        or alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    scale = FD004_SENSOR_SCALE_630V11[
        sensor
    ]

    target_std = float(
        alpha
        * scale
    )

    rng = np.random.default_rng(
        int(seed)
    )

    base_noise = rng.normal(
        loc=0.0,
        scale=1.0,
        size=len(df),
    )

    noise = (
        base_noise
        * target_std
    )

    corrupted = df.copy()

    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        noise
    )

    return (
        corrupted,
        noise,
        target_std
    )


def apply_fd004_drift_630v11(
    df,
    sensor,
    alpha,
    direction
):
    validate_sensor_fd004_630v11(
        df,
        sensor
    )

    if direction not in (
        -1,
        1
    ):
        raise ValueError(
            "direction must be -1 or +1."
        )

    alpha = float(
        alpha
    )

    if (
        not np.isfinite(alpha)
        or alpha <= 0.0
    ):
        raise ValueError(
            "alpha must be positive and finite."
        )

    final_drift = float(
        direction
        * alpha
        * FD004_SENSOR_SCALE_630V11[
            sensor
        ]
    )

    corrupted = df.copy()

    drift_values = np.zeros(
        len(corrupted),
        dtype=np.float64,
    )

    for _, group in (
        corrupted
        .groupby(
            "unit_id",
            sort=False
        )
    ):
        group_sorted = (
            group
            .sort_values(
                "cycle"
            )
        )

        positions = (
            group_sorted.index
            .to_numpy(
                dtype=np.int64
            )
        )

        n_rows = len(
            positions
        )

        if n_rows == 1:
            engine_drift = np.array(
                [
                    final_drift
                ],
                dtype=np.float64,
            )

        else:
            engine_drift = np.linspace(
                0.0,
                final_drift,
                num=n_rows,
                endpoint=True,
                dtype=np.float64,
            )

        drift_values[
            positions
        ] = engine_drift

    corrupted[
        sensor
    ] = (
        corrupted[
            sensor
        ].to_numpy(
            dtype=np.float64
        )
        +
        drift_values
    )

    return (
        corrupted,
        drift_values,
        final_drift
    )


def apply_fd004_dropout_630v11(
    df,
    sensor,
    dropout_rate,
    seed=42
):
    validate_sensor_fd004_630v11(
        df,
        sensor
    )

    dropout_rate = float(
        dropout_rate
    )

    if not (
        0.0
        <= dropout_rate
        < 1.0
    ):
        raise ValueError(
            "dropout_rate must satisfy 0 <= rate < 1."
        )

    if "regime_id" not in df.columns:
        raise ValueError(
            "regime_id is required for "
            "condition-aware imputation."
        )

    corrupted = df.copy()

    n_rows = len(
        corrupted
    )

    n_drop = int(
        round(
            dropout_rate
            * n_rows
        )
    )

    rng = np.random.default_rng(
        int(seed)
    )

    random_order = rng.permutation(
        n_rows
    )

    drop_positions = random_order[
        :n_drop
    ]

    drop_mask = np.zeros(
        n_rows,
        dtype=bool,
    )

    drop_mask[
        drop_positions
    ] = True

    sensor_col_pos = (
        corrupted.columns
        .get_loc(
            sensor
        )
    )

    corrupted.iloc[
        drop_positions,
        sensor_col_pos
    ] = np.nan

    regime_values = (
        corrupted[
            "regime_id"
        ]
        .to_numpy()
    )

    imputation_values = np.empty(
        n_rows,
        dtype=np.float64,
    )

    for position_630v11 in range(
        n_rows
    ):
        regime_630v11 = int(
            regime_values[
                position_630v11
            ]
        )

        key_630v11 = (
            regime_630v11,
            sensor
        )

        if (
            key_630v11
            not in FD004_DROPOUT_MEAN_630V11
        ):
            raise RuntimeError(
                "Missing FIT dropout mean for "
                f"regime={regime_630v11}, "
                f"sensor={sensor}."
            )

        imputation_values[
            position_630v11
        ] = (
            FD004_DROPOUT_MEAN_630V11[
                key_630v11
            ]
        )

    missing_positions = np.flatnonzero(
        drop_mask
    )

    corrupted.iloc[
        missing_positions,
        sensor_col_pos
    ] = (
        imputation_values[
            missing_positions
        ]
    )

    return (
        corrupted,
        drop_mask,
        imputation_values
    )


def apply_fd004_stuck_630v11(
    df,
    sensor,
    affected_fraction
):
    validate_sensor_fd004_630v11(
        df,
        sensor
    )

    affected_fraction = float(
        affected_fraction
    )

    if not (
        0.0
        < affected_fraction
        < 1.0
    ):
        raise ValueError(
            "affected_fraction must satisfy "
            "0 < fraction < 1."
        )

    corrupted = df.copy()

    affected_mask = np.zeros(
        len(corrupted),
        dtype=bool,
    )

    records_630v11 = []

    for unit_id_630v11, group_630v11 in (
        corrupted
        .groupby(
            "unit_id",
            sort=False
        )
    ):
        group_sorted_630v11 = (
            group_630v11
            .sort_values(
                "cycle"
            )
        )

        positions_630v11 = (
            group_sorted_630v11.index
            .to_numpy(
                dtype=np.int64
            )
        )

        n_rows_630v11 = len(
            positions_630v11
        )

        if n_rows_630v11 < 2:
            raise RuntimeError(
                "Stuck corruption requires "
                "at least two cycles per engine."
            )

        n_affected_630v11 = int(
            round(
                affected_fraction
                * n_rows_630v11
            )
        )

        n_affected_630v11 = max(
            1,
            n_affected_630v11
        )

        n_affected_630v11 = min(
            n_affected_630v11,
            n_rows_630v11 - 1,
        )

        onset_position_630v11 = (
            n_rows_630v11
            - n_affected_630v11
        )

        source_position_630v11 = (
            positions_630v11[
                onset_position_630v11 - 1
            ]
        )

        affected_positions_630v11 = (
            positions_630v11[
                onset_position_630v11:
            ]
        )

        freeze_value_630v11 = float(
            corrupted.loc[
                source_position_630v11,
                sensor
            ]
        )

        corrupted.loc[
            affected_positions_630v11,
            sensor
        ] = freeze_value_630v11

        affected_mask[
            affected_positions_630v11
        ] = True

        records_630v11.append(
            {
                "unit_id":
                    unit_id_630v11,

                "trajectory_length":
                    n_rows_630v11,

                "requested_fraction":
                    affected_fraction,

                "affected_count":
                    n_affected_630v11,

                "actual_fraction":
                    (
                        n_affected_630v11
                        / n_rows_630v11
                    ),

                "source_cycle":
                    float(
                        group_sorted_630v11
                        .iloc[
                            onset_position_630v11 - 1
                        ][
                            "cycle"
                        ]
                    ),

                "stuck_start_cycle":
                    float(
                        group_sorted_630v11
                        .iloc[
                            onset_position_630v11
                        ][
                            "cycle"
                        ]
                    ),

                "freeze_value":
                    freeze_value_630v11,
            }
        )

    stuck_info_630v11 = pd.DataFrame(
        records_630v11
    )

    return (
        corrupted,
        affected_mask,
        stuck_info_630v11
    )
"""


# Syntax validation before installation
compile(
    FD004_FIVE_OPERATOR_SOURCE_630V11,
    "<FD004_FIVE_OPERATOR_SOURCE_630V11>",
    "exec",
)

# Define functions, but DO NOT call them.
exec(
    compile(
        FD004_FIVE_OPERATOR_SOURCE_630V11,
        "<FD004_FIVE_OPERATOR_SOURCE_630V11>",
        "exec",
    ),
    globals(),
)


# ==================================================================================================
# 5) VERIFY EXACT SIGNATURE STRUCTURE
# ==================================================================================================

expected_signatures_630v11 = {
    "apply_fd004_bias_630v11":
        "(df, sensor, alpha, direction)",

    "apply_fd004_noise_630v11":
        "(df, sensor, alpha, seed=42)",

    "apply_fd004_drift_630v11":
        "(df, sensor, alpha, direction)",

    "apply_fd004_dropout_630v11":
        "(df, sensor, dropout_rate, seed=42)",

    "apply_fd004_stuck_630v11":
        "(df, sensor, affected_fraction)",
}

observed_signatures_630v11 = {}

for (
    fn_name_630v11,
    expected_sig_630v11,
) in expected_signatures_630v11.items():

    fn_630v11 = globals().get(
        fn_name_630v11
    )

    if (
        fn_630v11 is None
        or not callable(fn_630v11)
    ):
        raise RuntimeError(
            f"Adapted operator not callable: "
            f"{fn_name_630v11}"
        )

    observed_sig_630v11 = str(
        inspect.signature(
            fn_630v11
        )
    )

    observed_signatures_630v11[
        fn_name_630v11
    ] = observed_sig_630v11

    if (
        observed_sig_630v11
        != expected_sig_630v11
    ):
        raise RuntimeError(
            f"Signature mismatch for "
            f"{fn_name_630v11}: "
            f"{observed_sig_630v11} "
            f"!= {expected_sig_630v11}"
        )


# ==================================================================================================
# 6) LOCK OPERATOR REGISTRY
# ==================================================================================================

FD004_CORRUPTION_OPERATORS_630V11 = (
    MappingProxyType(
        {
            "bias":
                apply_fd004_bias_630v11,

            "noise":
                apply_fd004_noise_630v11,

            "drift":
                apply_fd004_drift_630v11,

            "dropout":
                apply_fd004_dropout_630v11,

            "stuck":
                apply_fd004_stuck_630v11,
        }
    )
)


# Exact recovered historical function fingerprints from STEP 6.30V-10.
FD004_HISTORICAL_OPERATOR_FUNCTION_SHA256_630V11 = (
    MappingProxyType(
        {
            "bias":
                "63de0768452ea7f4937546396a7621f062c46ab1345e120542cb51e214043f51",

            "noise":
                "d186d747071b7c190713f576d9d230ac44ac327c6a21e9320992f70775d4f57b",

            "drift":
                "761185996a49a699fab740337b7ddeb751ad31e753d30a1f7cf54935a5612ba9",

            "dropout":
                "b55016cf72cc19851a69c1fa16dee0ed84fc32afa0d0988682c96c368dcfd821",

            "stuck":
                "7e0d720c2fecef25b71a8aaeeecc7646c082504c338953d0586de54c490bb0d2",
        }
    )
)


# ==================================================================================================
# 7) IMPLEMENTATION / CONTRACT FINGERPRINTS
# ==================================================================================================

FD004_FIVE_OPERATOR_SOURCE_SHA256_630V11 = (
    hashlib.sha256(
        FD004_FIVE_OPERATOR_SOURCE_630V11
        .encode("utf-8")
    )
    .hexdigest()
)


FD004_FIVE_OPERATOR_CONTRACT_630V11 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-11",

    "historical_semantics_source_stage":
        "FD002_STEP_6.28GC",

    "historical_function_sha256":
        dict(
            FD004_HISTORICAL_OPERATOR_FUNCTION_SHA256_630V11
        ),

    "adapted_source_sha256":
        FD004_FIVE_OPERATOR_SOURCE_SHA256_630V11,

    "sensor_scale_authority_sha256":
        str(
            FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T
        ),

    "dropout_imputation_authority_sha256":
        str(
            FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T
        ),

    "sensor_count":
        len(
            FD004_SENSOR_COLS_630V11
        ),

    "scale_entries":
        len(
            FD004_SENSOR_SCALE_630V11
        ),

    "dropout_mean_entries":
        len(
            FD004_DROPOUT_MEAN_630V11
        ),

    "signatures":
        dict(
            observed_signatures_630v11
        ),

    "operators_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11 = (
    hashlib.sha256(
        json.dumps(
            FD004_FIVE_OPERATOR_CONTRACT_630V11,
            sort_keys=True,
            separators=(",", ":"),
        )
        .encode("utf-8")
    )
    .hexdigest()
)


# ==================================================================================================
# 8) LOCK FLAGS
# ==================================================================================================

FD004_FIVE_OPERATOR_IMPLEMENTATION_LOCKED_630V11 = True

FD004_CORRUPTION_EXECUTED_630V11 = False
FD004_OPERATOR_EXECUTED_630V11 = False

FD004_MODEL_INFERENCE_PERFORMED_630V11 = False
FD004_MC_EXECUTION_PERFORMED_630V11 = False

FD004_DEV_TARGETS_ERRORS_USED_630V11 = False

FD004_THRESHOLD_SELECTED_630V11 = False
FD004_POLICY_SELECTED_630V11 = False

FD004_TEST_CONTENTS_READ_630V11 = False
FD004_RUL_CONTENTS_READ_630V11 = False

all_checks_passed_630v11 = True


# ==================================================================================================
# 9) REPORT
# ==================================================================================================

print("=" * 176)

print(
    "STEP 6.30V-11 — FD004 EXACT-SEMANTICS FIVE-FAMILY "
    "CORRUPTION-OPERATOR IMPLEMENTATION LOCK"
)

print("=" * 176)

print(
    "Operators instantiated            :",
    True,
)

print(
    "Operators executed                :",
    False,
)

print(
    "Corruption executed               :",
    False,
)

print(
    "Model inference performed         :",
    False,
)

print(
    "MC execution performed            :",
    False,
)

print(
    "DEV targets/errors used           :",
    False,
)

print(
    "Threshold selected                :",
    False,
)

print(
    "Policy selected                   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print(
    "\n--- FD004 REFERENCE BINDING ---"
)

print(
    "Physical sensors                  :",
    len(
        FD004_SENSOR_COLS_630V11
    ),
)

print(
    "Magnitude-scale entries           :",
    len(
        FD004_SENSOR_SCALE_630V11
    ),
)

print(
    "Dropout regime×sensor means       :",
    len(
        FD004_DROPOUT_MEAN_630V11
    ),
)

print(
    "Scale authority SHA-256           :",
    FD004_CORRUPTION_SCALE_REFERENCE_SHA256_630T,
)

print(
    "Dropout authority SHA-256         :",
    FD004_DROPOUT_IMPUTATION_REFERENCE_SHA256_630T,
)


print(
    "\n--- LOCKED OPERATOR SIGNATURES ---"
)

for (
    family_630v11,
    fn_630v11,
) in FD004_CORRUPTION_OPERATORS_630V11.items():

    print(
        f"{family_630v11:8s}: "
        f"{fn_630v11.__name__}"
        f"{inspect.signature(fn_630v11)}"
    )


print(
    "\n--- HISTORICAL FUNCTION AUTHORITIES ---"
)

for (
    family_630v11,
    sha_630v11,
) in (
    FD004_HISTORICAL_OPERATOR_FUNCTION_SHA256_630V11
    .items()
):

    print(
        f"{family_630v11:8s}: "
        f"{sha_630v11}"
    )


print(
    "\n--- IMPLEMENTATION FINGERPRINTS ---"
)

print(
    "Adapted operator source SHA-256   :",
    FD004_FIVE_OPERATOR_SOURCE_SHA256_630V11,
)

print(
    "Five-operator contract SHA-256    :",
    FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11,
)


print(
    "\nPASS — the FD004 bias/noise/drift/dropout/stuck "
    "operators are now instantiated from the exact recovered "
    "historical semantics and bound only to frozen FD004 "
    "FIT-derived corruption references."
)

print(
    "No operator has been executed yet."
)

print(
    "Next: perform structural family-by-family validation, "
    "including exact reproduction of the already-locked bias "
    "smoke case, before any 381-scenario corruption run."
)

In [ ]:
# STEP 6.30V-12 — FD004 FIVE-FAMILY STRUCTURAL OPERATOR VALIDATION + SMOKE LOCK

import numpy as np
import pandas as pd
import hashlib
import json

# 0) Preconditions
required_630v12 = (
    "all_checks_passed_630v11",
    "FD004_CORRUPTION_OPERATORS_630V11",
    "FD004_SENSOR_SCALE_630V11",
    "FD004_DROPOUT_MEAN_630V11",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",
)

missing_630v12 = [
    n for n in required_630v12
    if n not in globals()
]

if missing_630v12:
    raise RuntimeError(
        "Missing STEP 6.30V-12 authority object(s): "
        + ", ".join(missing_630v12)
    )

if not bool(all_checks_passed_630v11):
    raise RuntimeError(
        "STEP 6.30V-11 must pass before STEP 6.30V-12."
    )

operators_630v12 = FD004_CORRUPTION_OPERATORS_630V11

expected_families_630v12 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

if tuple(operators_630v12.keys()) != expected_families_630v12:
    raise RuntimeError(
        "Frozen FD004 operator registry family order/content changed."
    )

manifest_630v12 = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .copy()
)

dev_raw_630v12 = FD004_DEV_RAW_630C

dev_regime_ids_630v12 = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

if len(dev_raw_630v12) != len(dev_regime_ids_630v12):
    raise RuntimeError(
        "DEV raw/regime-ID lengths differ."
    )

expected_index_630v12 = np.arange(
    len(dev_raw_630v12),
    dtype=np.int64,
)

observed_index_630v12 = (
    dev_raw_630v12.index
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    observed_index_630v12,
    expected_index_630v12,
):
    raise RuntimeError(
        "FD004 DEV raw index is not exact 0..N-1; "
        "historical positional semantics would be ambiguous."
    )

dev_raw_snapshot_630v12 = (
    dev_raw_630v12.copy(deep=True)
)

dev_regime_snapshot_630v12 = (
    dev_regime_ids_630v12.copy()
)


# Helpers
def _sha_array_630v12(x):
    arr = np.asarray(x)
    carr = np.ascontiguousarray(arr)

    h = hashlib.sha256()
    h.update(str(carr.dtype).encode("utf-8"))
    h.update(str(tuple(carr.shape)).encode("utf-8"))
    h.update(carr.tobytes(order="C"))

    return h.hexdigest()


def _row_for_family_630v12(family):
    family_rows = manifest_630v12.loc[
        manifest_630v12["family"].astype(str)
        == str(family)
    ].copy()

    if len(family_rows) == 0:
        raise RuntimeError(
            f"No executable scenario found for family={family}."
        )

    family_rows = family_rows.sort_values(
        "scenario_number",
        kind="mergesort",
    )

    return family_rows.iloc[0]


# Validation-scenario selection
bias_rows_630v12 = manifest_630v12.loc[
    manifest_630v12["scenario_id"].astype(str)
    == str(FD004_SMOKE_SCENARIO_ID_630V2)
].copy()

if len(bias_rows_630v12) != 1:
    raise RuntimeError(
        "Frozen bias smoke scenario does not resolve to "
        "exactly one executable-manifest row."
    )

scenario_rows_630v12 = {
    "bias":
        bias_rows_630v12.iloc[0],

    "noise":
        _row_for_family_630v12("noise"),

    "drift":
        _row_for_family_630v12("drift"),

    "dropout":
        _row_for_family_630v12("dropout"),

    "stuck":
        _row_for_family_630v12("stuck"),
}


# ============================================================
# 1) BIAS — exact frozen smoke reproduction
# ============================================================

bias_row_630v12 = scenario_rows_630v12["bias"]

bias_sensor_630v12 = str(
    bias_row_630v12["sensor"]
)

bias_alpha_630v12 = float(
    bias_row_630v12["alpha"]
)

bias_direction_630v12 = int(
    bias_row_630v12["direction"]
)

(
    bias_corrupted_630v12,
    bias_amount_630v12,
) = operators_630v12["bias"](
    dev_raw_630v12,
    bias_sensor_630v12,
    bias_alpha_630v12,
    bias_direction_630v12,
)

bias_expected_amount_630v12 = float(
    bias_direction_630v12
    * bias_alpha_630v12
    * float(
        FD004_SENSOR_SCALE_630V11[
            bias_sensor_630v12
        ]
    )
)

bias_amount_exact_630v12 = bool(
    bias_amount_630v12
    == bias_expected_amount_630v12
)

bias_raw_exact_630v12 = bool(
    tuple(bias_corrupted_630v12.columns)
    == tuple(
        FD004_SMOKE_CORRUPTED_RAW_630V2.columns
    )
    and np.array_equal(
        bias_corrupted_630v12.to_numpy(),
        FD004_SMOKE_CORRUPTED_RAW_630V2.to_numpy(),
        equal_nan=True,
    )
)

if not bias_amount_exact_630v12:
    raise RuntimeError(
        "Bias amount failed exact historical-semantics check."
    )

if not bias_raw_exact_630v12:
    raise RuntimeError(
        "Bias operator failed exact frozen smoke reproduction."
    )


# ============================================================
# 2) NOISE — exact seeded-vector semantics
# ============================================================

noise_row_630v12 = scenario_rows_630v12["noise"]

noise_sensor_630v12 = str(
    noise_row_630v12["sensor"]
)

noise_alpha_630v12 = float(
    noise_row_630v12["alpha"]
)

noise_seed_raw_630v12 = (
    noise_row_630v12["seed"]
)

if pd.isna(noise_seed_raw_630v12):
    raise RuntimeError(
        "Selected noise scenario has no seed."
    )

noise_seed_630v12 = int(
    noise_seed_raw_630v12
)

(
    noise_corrupted_630v12,
    noise_vector_630v12,
    noise_target_std_630v12,
) = operators_630v12["noise"](
    dev_raw_630v12,
    noise_sensor_630v12,
    noise_alpha_630v12,
    noise_seed_630v12,
)

noise_expected_std_630v12 = float(
    noise_alpha_630v12
    * float(
        FD004_SENSOR_SCALE_630V11[
            noise_sensor_630v12
        ]
    )
)

noise_rng_630v12 = np.random.default_rng(
    noise_seed_630v12
)

noise_expected_vector_630v12 = (
    noise_rng_630v12.normal(
        loc=0.0,
        scale=1.0,
        size=len(dev_raw_630v12),
    )
    * noise_expected_std_630v12
)

noise_vector_exact_630v12 = bool(
    np.array_equal(
        np.asarray(
            noise_vector_630v12,
            dtype=np.float64,
        ),
        np.asarray(
            noise_expected_vector_630v12,
            dtype=np.float64,
        ),
    )
)

noise_std_exact_630v12 = bool(
    float(noise_target_std_630v12)
    == noise_expected_std_630v12
)

noise_expected_sensor_630v12 = (
    dev_raw_630v12[
        noise_sensor_630v12
    ].to_numpy(dtype=np.float64)
    + noise_expected_vector_630v12
)

noise_sensor_exact_630v12 = bool(
    np.array_equal(
        noise_corrupted_630v12[
            noise_sensor_630v12
        ].to_numpy(dtype=np.float64),
        noise_expected_sensor_630v12,
    )
)

noise_other_cols_630v12 = [
    c
    for c in dev_raw_630v12.columns
    if c != noise_sensor_630v12
]

noise_other_unchanged_630v12 = bool(
    np.array_equal(
        noise_corrupted_630v12[
            noise_other_cols_630v12
        ].to_numpy(),
        dev_raw_630v12[
            noise_other_cols_630v12
        ].to_numpy(),
        equal_nan=True,
    )
)

if not all(
    (
        noise_vector_exact_630v12,
        noise_std_exact_630v12,
        noise_sensor_exact_630v12,
        noise_other_unchanged_630v12,
    )
):
    raise RuntimeError(
        "Noise operator structural validation failed."
    )


# ============================================================
# 3) DRIFT — exact per-engine linear-ramp semantics
# ============================================================

drift_row_630v12 = scenario_rows_630v12["drift"]

drift_sensor_630v12 = str(
    drift_row_630v12["sensor"]
)

drift_alpha_630v12 = float(
    drift_row_630v12["alpha"]
)

drift_direction_630v12 = int(
    drift_row_630v12["direction"]
)

(
    drift_corrupted_630v12,
    drift_vector_630v12,
    drift_final_630v12,
) = operators_630v12["drift"](
    dev_raw_630v12,
    drift_sensor_630v12,
    drift_alpha_630v12,
    drift_direction_630v12,
)

drift_expected_final_630v12 = float(
    drift_direction_630v12
    * drift_alpha_630v12
    * float(
        FD004_SENSOR_SCALE_630V11[
            drift_sensor_630v12
        ]
    )
)

drift_expected_vector_630v12 = np.zeros(
    len(dev_raw_630v12),
    dtype=np.float64,
)

for _, group_630v12 in (
    dev_raw_630v12.groupby(
        "unit_id",
        sort=False,
    )
):
    group_sorted_630v12 = (
        group_630v12.sort_values(
            "cycle"
        )
    )

    positions_630v12 = (
        group_sorted_630v12.index
        .to_numpy(dtype=np.int64)
    )

    n_rows_630v12 = len(
        positions_630v12
    )

    if n_rows_630v12 == 1:
        engine_drift_630v12 = np.array(
            [
                drift_expected_final_630v12
            ],
            dtype=np.float64,
        )
    else:
        engine_drift_630v12 = np.linspace(
            0.0,
            drift_expected_final_630v12,
            num=n_rows_630v12,
            endpoint=True,
            dtype=np.float64,
        )

    drift_expected_vector_630v12[
        positions_630v12
    ] = engine_drift_630v12

drift_vector_exact_630v12 = bool(
    np.array_equal(
        np.asarray(
            drift_vector_630v12,
            dtype=np.float64,
        ),
        drift_expected_vector_630v12,
    )
)

drift_final_exact_630v12 = bool(
    float(drift_final_630v12)
    == drift_expected_final_630v12
)

drift_expected_sensor_630v12 = (
    dev_raw_630v12[
        drift_sensor_630v12
    ].to_numpy(dtype=np.float64)
    + drift_expected_vector_630v12
)

drift_sensor_exact_630v12 = bool(
    np.array_equal(
        drift_corrupted_630v12[
            drift_sensor_630v12
        ].to_numpy(dtype=np.float64),
        drift_expected_sensor_630v12,
    )
)

drift_other_cols_630v12 = [
    c
    for c in dev_raw_630v12.columns
    if c != drift_sensor_630v12
]

drift_other_unchanged_630v12 = bool(
    np.array_equal(
        drift_corrupted_630v12[
            drift_other_cols_630v12
        ].to_numpy(),
        dev_raw_630v12[
            drift_other_cols_630v12
        ].to_numpy(),
        equal_nan=True,
    )
)

if not all(
    (
        drift_vector_exact_630v12,
        drift_final_exact_630v12,
        drift_sensor_exact_630v12,
        drift_other_unchanged_630v12,
    )
):
    raise RuntimeError(
        "Drift operator structural validation failed."
    )


# ============================================================
# 4) DROPOUT — exact seed + FIT regime-mean imputation
# ============================================================

dropout_row_630v12 = (
    scenario_rows_630v12["dropout"]
)

dropout_sensor_630v12 = str(
    dropout_row_630v12["sensor"]
)

dropout_fraction_630v12 = float(
    dropout_row_630v12[
        "requested_fraction"
    ]
)

dropout_seed_raw_630v12 = (
    dropout_row_630v12["seed"]
)

if pd.isna(dropout_seed_raw_630v12):
    raise RuntimeError(
        "Selected dropout scenario has no seed."
    )

dropout_seed_630v12 = int(
    dropout_seed_raw_630v12
)

dropout_input_630v12 = (
    dev_raw_630v12.copy(deep=True)
)

dropout_input_630v12[
    "regime_id"
] = dev_regime_ids_630v12

(
    dropout_corrupted_630v12,
    dropout_mask_630v12,
    dropout_imputation_values_630v12,
) = operators_630v12["dropout"](
    dropout_input_630v12,
    dropout_sensor_630v12,
    dropout_fraction_630v12,
    dropout_seed_630v12,
)

dropout_n_630v12 = len(
    dropout_input_630v12
)

dropout_expected_count_630v12 = int(
    round(
        dropout_fraction_630v12
        * dropout_n_630v12
    )
)

dropout_rng_630v12 = (
    np.random.default_rng(
        dropout_seed_630v12
    )
)

dropout_expected_positions_630v12 = (
    dropout_rng_630v12.permutation(
        dropout_n_630v12
    )[
        :dropout_expected_count_630v12
    ]
)

dropout_expected_mask_630v12 = np.zeros(
    dropout_n_630v12,
    dtype=bool,
)

dropout_expected_mask_630v12[
    dropout_expected_positions_630v12
] = True

dropout_expected_imputation_630v12 = np.asarray(
    [
        float(
            FD004_DROPOUT_MEAN_630V11[
                (
                    int(regime_630v12),
                    dropout_sensor_630v12,
                )
            ]
        )
        for regime_630v12
        in dev_regime_ids_630v12
    ],
    dtype=np.float64,
)

dropout_mask_exact_630v12 = bool(
    np.array_equal(
        np.asarray(
            dropout_mask_630v12,
            dtype=bool,
        ),
        dropout_expected_mask_630v12,
    )
)

dropout_imputation_exact_630v12 = bool(
    np.array_equal(
        np.asarray(
            dropout_imputation_values_630v12,
            dtype=np.float64,
        ),
        dropout_expected_imputation_630v12,
    )
)

dropout_expected_sensor_630v12 = (
    dropout_input_630v12[
        dropout_sensor_630v12
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

dropout_expected_sensor_630v12[
    dropout_expected_mask_630v12
] = (
    dropout_expected_imputation_630v12[
        dropout_expected_mask_630v12
    ]
)

dropout_sensor_exact_630v12 = bool(
    np.array_equal(
        dropout_corrupted_630v12[
            dropout_sensor_630v12
        ].to_numpy(dtype=np.float64),
        dropout_expected_sensor_630v12,
    )
)

dropout_other_cols_630v12 = [
    c
    for c in dropout_input_630v12.columns
    if c != dropout_sensor_630v12
]

dropout_other_unchanged_630v12 = bool(
    np.array_equal(
        dropout_corrupted_630v12[
            dropout_other_cols_630v12
        ].to_numpy(),
        dropout_input_630v12[
            dropout_other_cols_630v12
        ].to_numpy(),
        equal_nan=True,
    )
)

dropout_all_finite_630v12 = bool(
    np.all(
        np.isfinite(
            dropout_corrupted_630v12[
                dropout_sensor_630v12
            ].to_numpy(dtype=np.float64)
        )
    )
)

if not all(
    (
        dropout_mask_exact_630v12,
        dropout_imputation_exact_630v12,
        dropout_sensor_exact_630v12,
        dropout_other_unchanged_630v12,
        dropout_all_finite_630v12,
    )
):
    raise RuntimeError(
        "Dropout operator structural validation failed."
    )


# ============================================================
# 5) STUCK — exact terminal-segment freeze semantics
# ============================================================

stuck_row_630v12 = (
    scenario_rows_630v12["stuck"]
)

stuck_sensor_630v12 = str(
    stuck_row_630v12["sensor"]
)

stuck_fraction_630v12 = float(
    stuck_row_630v12[
        "requested_fraction"
    ]
)

(
    stuck_corrupted_630v12,
    stuck_mask_630v12,
    stuck_info_630v12,
) = operators_630v12["stuck"](
    dev_raw_630v12,
    stuck_sensor_630v12,
    stuck_fraction_630v12,
)

stuck_expected_mask_630v12 = np.zeros(
    len(dev_raw_630v12),
    dtype=bool,
)

stuck_expected_sensor_630v12 = (
    dev_raw_630v12[
        stuck_sensor_630v12
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

stuck_expected_records_630v12 = []

for (
    unit_id_630v12,
    group_630v12,
) in dev_raw_630v12.groupby(
    "unit_id",
    sort=False,
):
    group_sorted_630v12 = (
        group_630v12.sort_values(
            "cycle"
        )
    )

    positions_630v12 = (
        group_sorted_630v12.index
        .to_numpy(dtype=np.int64)
    )

    n_rows_630v12 = len(
        positions_630v12
    )

    if n_rows_630v12 < 2:
        raise RuntimeError(
            "DEV engine unexpectedly has fewer than two cycles."
        )

    n_affected_630v12 = int(
        round(
            stuck_fraction_630v12
            * n_rows_630v12
        )
    )

    n_affected_630v12 = max(
        1,
        n_affected_630v12,
    )

    n_affected_630v12 = min(
        n_affected_630v12,
        n_rows_630v12 - 1,
    )

    onset_630v12 = (
        n_rows_630v12
        - n_affected_630v12
    )

    source_position_630v12 = (
        positions_630v12[
            onset_630v12 - 1
        ]
    )

    affected_positions_630v12 = (
        positions_630v12[
            onset_630v12:
        ]
    )

    freeze_value_630v12 = float(
        dev_raw_630v12.loc[
            source_position_630v12,
            stuck_sensor_630v12,
        ]
    )

    stuck_expected_sensor_630v12[
        affected_positions_630v12
    ] = freeze_value_630v12

    stuck_expected_mask_630v12[
        affected_positions_630v12
    ] = True

    stuck_expected_records_630v12.append(
        {
            "unit_id":
                unit_id_630v12,

            "trajectory_length":
                n_rows_630v12,

            "requested_fraction":
                stuck_fraction_630v12,

            "affected_count":
                n_affected_630v12,

            "actual_fraction":
                (
                    n_affected_630v12
                    / n_rows_630v12
                ),

            "source_cycle":
                float(
                    group_sorted_630v12.iloc[
                        onset_630v12 - 1
                    ]["cycle"]
                ),

            "stuck_start_cycle":
                float(
                    group_sorted_630v12.iloc[
                        onset_630v12
                    ]["cycle"]
                ),

            "freeze_value":
                freeze_value_630v12,
        }
    )

stuck_expected_info_630v12 = pd.DataFrame(
    stuck_expected_records_630v12
)

stuck_mask_exact_630v12 = bool(
    np.array_equal(
        np.asarray(
            stuck_mask_630v12,
            dtype=bool,
        ),
        stuck_expected_mask_630v12,
    )
)

stuck_sensor_exact_630v12 = bool(
    np.array_equal(
        stuck_corrupted_630v12[
            stuck_sensor_630v12
        ].to_numpy(dtype=np.float64),
        stuck_expected_sensor_630v12,
    )
)

stuck_other_cols_630v12 = [
    c
    for c in dev_raw_630v12.columns
    if c != stuck_sensor_630v12
]

stuck_other_unchanged_630v12 = bool(
    np.array_equal(
        stuck_corrupted_630v12[
            stuck_other_cols_630v12
        ].to_numpy(),
        dev_raw_630v12[
            stuck_other_cols_630v12
        ].to_numpy(),
        equal_nan=True,
    )
)

stuck_info_exact_630v12 = bool(
    tuple(stuck_info_630v12.columns)
    == tuple(stuck_expected_info_630v12.columns)
    and np.array_equal(
        stuck_info_630v12.to_numpy(),
        stuck_expected_info_630v12.to_numpy(),
        equal_nan=True,
    )
)

if not all(
    (
        stuck_mask_exact_630v12,
        stuck_sensor_exact_630v12,
        stuck_other_unchanged_630v12,
        stuck_info_exact_630v12,
    )
):
    raise RuntimeError(
        "Stuck operator structural validation failed."
    )


# ============================================================
# 6) Confirm frozen source authorities were not mutated
# ============================================================

dev_raw_unchanged_630v12 = bool(
    dev_raw_630v12.equals(
        dev_raw_snapshot_630v12
    )
)

dev_regime_unchanged_630v12 = bool(
    np.array_equal(
        dev_regime_ids_630v12,
        dev_regime_snapshot_630v12,
    )
)

if not dev_raw_unchanged_630v12:
    raise RuntimeError(
        "FD004 DEV raw authority was mutated."
    )

if not dev_regime_unchanged_630v12:
    raise RuntimeError(
        "FD004 DEV regime-ID authority was mutated."
    )


# ============================================================
# 7) Freeze validation evidence
# ============================================================

FD004_OPERATOR_VALIDATION_SCENARIOS_630V12 = {
    family_630v12: {
        "scenario_id":
            str(
                row_630v12["scenario_id"]
            ),

        "scenario_number":
            int(
                row_630v12["scenario_number"]
            ),

        "sensor":
            str(
                row_630v12["sensor"]
            ),

        "severity":
            str(
                row_630v12["severity"]
            ),
    }
    for (
        family_630v12,
        row_630v12,
    ) in scenario_rows_630v12.items()
}


FD004_OPERATOR_VALIDATION_HASHES_630V12 = {
    "bias_corrupted_sensor":
        _sha_array_630v12(
            bias_corrupted_630v12[
                bias_sensor_630v12
            ].to_numpy(dtype=np.float64)
        ),

    "noise_vector":
        _sha_array_630v12(
            noise_vector_630v12
        ),

    "drift_vector":
        _sha_array_630v12(
            drift_vector_630v12
        ),

    "dropout_mask":
        _sha_array_630v12(
            dropout_mask_630v12
        ),

    "dropout_imputation_values":
        _sha_array_630v12(
            dropout_imputation_values_630v12
        ),

    "stuck_mask":
        _sha_array_630v12(
            stuck_mask_630v12
        ),

    "stuck_corrupted_sensor":
        _sha_array_630v12(
            stuck_corrupted_630v12[
                stuck_sensor_630v12
            ].to_numpy(dtype=np.float64)
        ),
}


FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_630V12 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-12",

    "operator_contract_sha256":
        str(
            FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11
        ),

    "full_execution_contract_sha256":
        str(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6
        ),

    "validation_scenarios":
        FD004_OPERATOR_VALIDATION_SCENARIOS_630V12,

    "bias_exact_frozen_smoke_reproduction":
        bool(
            bias_raw_exact_630v12
        ),

    "noise_seeded_vector_exact":
        bool(
            noise_vector_exact_630v12
        ),

    "drift_engine_ramps_exact":
        bool(
            drift_vector_exact_630v12
        ),

    "dropout_seeded_mask_and_fit_imputation_exact":
        bool(
            dropout_mask_exact_630v12
            and dropout_imputation_exact_630v12
        ),

    "stuck_terminal_segment_semantics_exact":
        bool(
            stuck_mask_exact_630v12
            and stuck_info_exact_630v12
        ),

    "dev_raw_authority_unchanged":
        bool(
            dev_raw_unchanged_630v12
        ),

    "dev_regime_authority_unchanged":
        bool(
            dev_regime_unchanged_630v12
        ),

    "validation_hashes":
        dict(
            FD004_OPERATOR_VALIDATION_HASHES_630V12
        ),

    "full_381_scenario_run_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_SHA256_630V12 = (
    hashlib.sha256(
        json.dumps(
            FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_630V12,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_LOCKED_630V12 = True

FD004_STRUCTURAL_CORRUPTION_VALIDATIONS_EXECUTED_630V12 = True

FD004_FULL_CORRUPTION_EXECUTED_630V12 = False

FD004_MODEL_INFERENCE_PERFORMED_630V12 = False
FD004_MC_EXECUTION_PERFORMED_630V12 = False

FD004_DEV_TARGETS_ERRORS_USED_630V12 = False

FD004_THRESHOLD_SELECTED_630V12 = False
FD004_POLICY_SELECTED_630V12 = False

FD004_TEST_CONTENTS_READ_630V12 = False
FD004_RUL_CONTENTS_READ_630V12 = False

all_checks_passed_630v12 = True


# ============================================================
# 8) Report
# ============================================================

print("=" * 176)

print(
    "STEP 6.30V-12 — FD004 FIVE-FAMILY STRUCTURAL "
    "OPERATOR VALIDATION + SMOKE LOCK"
)

print("=" * 176)

print(
    "Structural operator executions    :",
    5,
)

print(
    "Full 381-scenario run executed    :",
    False,
)

print(
    "Model inference performed         :",
    False,
)

print(
    "MC execution performed            :",
    False,
)

print(
    "DEV targets/errors used           :",
    False,
)

print(
    "Threshold selected                :",
    False,
)

print(
    "Policy selected                   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print("\n--- VALIDATION SCENARIOS ---")

for family_630v12 in expected_families_630v12:
    info_630v12 = (
        FD004_OPERATOR_VALIDATION_SCENARIOS_630V12[
            family_630v12
        ]
    )

    print(
        f"{family_630v12:8s}: "
        f"{info_630v12['scenario_id']} | "
        f"#{info_630v12['scenario_number']} | "
        f"{info_630v12['sensor']} | "
        f"{info_630v12['severity']}"
    )


print("\n--- BIAS ---")

print(
    "Frozen smoke raw exact            :",
    bias_raw_exact_630v12,
)

print(
    "Bias amount exact                 :",
    bias_amount_exact_630v12,
)


print("\n--- NOISE ---")

print(
    "Seeded noise vector exact         :",
    noise_vector_exact_630v12,
)

print(
    "Target STD exact                  :",
    noise_std_exact_630v12,
)

print(
    "Corrupted sensor exact            :",
    noise_sensor_exact_630v12,
)

print(
    "Non-target columns unchanged      :",
    noise_other_unchanged_630v12,
)


print("\n--- DRIFT ---")

print(
    "Per-engine ramp vector exact      :",
    drift_vector_exact_630v12,
)

print(
    "Final drift exact                 :",
    drift_final_exact_630v12,
)

print(
    "Corrupted sensor exact            :",
    drift_sensor_exact_630v12,
)

print(
    "Non-target columns unchanged      :",
    drift_other_unchanged_630v12,
)


print("\n--- DROPOUT ---")

print(
    "Seeded drop mask exact            :",
    dropout_mask_exact_630v12,
)

print(
    "FIT imputation vector exact       :",
    dropout_imputation_exact_630v12,
)

print(
    "Corrupted sensor exact            :",
    dropout_sensor_exact_630v12,
)

print(
    "Non-target columns unchanged      :",
    dropout_other_unchanged_630v12,
)

print(
    "Corrupted sensor finite           :",
    dropout_all_finite_630v12,
)

print(
    "Dropped rows                     :",
    int(
        np.asarray(
            dropout_mask_630v12,
            dtype=bool,
        ).sum()
    ),
)


print("\n--- STUCK ---")

print(
    "Affected mask exact               :",
    stuck_mask_exact_630v12,
)

print(
    "Terminal freeze sensor exact      :",
    stuck_sensor_exact_630v12,
)

print(
    "Non-target columns unchanged      :",
    stuck_other_unchanged_630v12,
)

print(
    "Per-engine stuck info exact       :",
    stuck_info_exact_630v12,
)

print(
    "DEV engines represented           :",
    len(
        stuck_info_630v12
    ),
)


print("\n--- AUTHORITY IMMUTABILITY ---")

print(
    "DEV raw authority unchanged       :",
    dev_raw_unchanged_630v12,
)

print(
    "DEV regime authority unchanged    :",
    dev_regime_unchanged_630v12,
)


print("\n--- CONTRACT FINGERPRINT ---")

print(
    "Structural validation SHA-256     :",
    FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_SHA256_630V12,
)


print(
    "\nPASS — all five FD004 corruption operators satisfy the "
    "exact recovered historical semantics on prospectively selected "
    "admissible structural validation scenarios."
)

print(
    "The bias operator also reproduces the already-frozen smoke raw "
    "matrix exactly. No model/reliability scoring or DEV outcome "
    "information was used."
)

print(
    "Next: begin the frozen 381-scenario label-free corruption "
    "execution in family blocks, retaining only prediction, MC STD, "
    "final risk, and required hashes according to STEP 6.30V-6."
)

In [ ]:
# STEP 6.30V-12R1 — FD004 ROW-INDEX / POSITIONAL-SEMANTICS AUTHORITY AUDIT

import numpy as np
import pandas as pd
import hashlib
import json

required_630v12r1 = (
    "all_checks_passed_630v11",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_NORMALIZED_630F",
    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_SMOKE_NORMALIZED_630V2",
)

missing_630v12r1 = [
    n for n in required_630v12r1
    if n not in globals()
]

if missing_630v12r1:
    raise RuntimeError(
        "Missing STEP 6.30V-12R1 authority object(s): "
        + ", ".join(missing_630v12r1)
    )

if not bool(all_checks_passed_630v11):
    raise RuntimeError(
        "STEP 6.30V-11 must remain passed."
    )

raw_630v12r1 = FD004_DEV_RAW_630C
norm_630v12r1 = FD004_DEV_NORMALIZED_630F
smoke_raw_630v12r1 = FD004_SMOKE_CORRUPTED_RAW_630V2
smoke_norm_630v12r1 = FD004_SMOKE_NORMALIZED_630V2

for name_630v12r1, df_630v12r1 in (
    ("FD004_DEV_RAW_630C", raw_630v12r1),
    ("FD004_DEV_NORMALIZED_630F", norm_630v12r1),
    ("FD004_SMOKE_CORRUPTED_RAW_630V2", smoke_raw_630v12r1),
    ("FD004_SMOKE_NORMALIZED_630V2", smoke_norm_630v12r1),
):
    if not isinstance(df_630v12r1, pd.DataFrame):
        raise RuntimeError(
            f"{name_630v12r1} is not a pandas DataFrame."
        )

n_630v12r1 = len(raw_630v12r1)

if n_630v12r1 != 12274:
    raise RuntimeError(
        f"Unexpected FD004 DEV raw row count: {n_630v12r1}"
    )

raw_index_630v12r1 = raw_630v12r1.index.to_numpy()
range_index_630v12r1 = np.arange(
    n_630v12r1,
    dtype=np.int64,
)

raw_index_is_range_630v12r1 = bool(
    np.array_equal(
        np.asarray(
            raw_index_630v12r1,
            dtype=np.int64,
        ),
        range_index_630v12r1,
    )
)

raw_index_unique_630v12r1 = bool(
    raw_630v12r1.index.is_unique
)

raw_index_monotonic_630v12r1 = bool(
    raw_630v12r1.index.is_monotonic_increasing
)

bridge_index_checks_630v12r1 = {
    "dev_normalized_matches_dev_raw":
        bool(
            norm_630v12r1.index.equals(
                raw_630v12r1.index
            )
        ),

    "smoke_raw_matches_dev_raw":
        bool(
            smoke_raw_630v12r1.index.equals(
                raw_630v12r1.index
            )
        ),

    "smoke_normalized_matches_dev_raw":
        bool(
            smoke_norm_630v12r1.index.equals(
                raw_630v12r1.index
            )
        ),
}

if not all(
    bridge_index_checks_630v12r1.values()
):
    raise RuntimeError(
        "Frozen FD004 preprocessing/smoke bridge does not preserve "
        "the DEV raw row-index authority exactly."
    )

required_identity_cols_630v12r1 = (
    "unit_id",
    "cycle",
)

for col_630v12r1 in required_identity_cols_630v12r1:
    if col_630v12r1 not in raw_630v12r1.columns:
        raise RuntimeError(
            f"Missing identity column: {col_630v12r1}"
        )

unit_630v12r1 = (
    raw_630v12r1[
        "unit_id"
    ].to_numpy()
)

cycle_630v12r1 = (
    raw_630v12r1[
        "cycle"
    ].to_numpy()
)

lex_order_630v12r1 = np.lexsort(
    (
        np.asarray(cycle_630v12r1),
        np.asarray(unit_630v12r1),
    )
)

row_order_is_unit_cycle_sorted_630v12r1 = bool(
    np.array_equal(
        lex_order_630v12r1,
        range_index_630v12r1,
    )
)

per_engine_cycle_sorted_630v12r1 = True
per_engine_rows_contiguous_630v12r1 = True

positions_630v12r1 = np.arange(
    n_630v12r1,
    dtype=np.int64,
)

tmp_identity_630v12r1 = pd.DataFrame(
    {
        "unit_id":
            unit_630v12r1,

        "cycle":
            cycle_630v12r1,

        "_row_position":
            positions_630v12r1,
    }
)

for _, grp_630v12r1 in (
    tmp_identity_630v12r1.groupby(
        "unit_id",
        sort=False,
    )
):
    pos_630v12r1 = (
        grp_630v12r1[
            "_row_position"
        ].to_numpy(dtype=np.int64)
    )

    cyc_630v12r1 = (
        grp_630v12r1[
            "cycle"
        ].to_numpy()
    )

    if len(cyc_630v12r1) > 1:
        if not bool(
            np.all(
                np.diff(
                    cyc_630v12r1
                ) > 0
            )
        ):
            per_engine_cycle_sorted_630v12r1 = False

        if not bool(
            np.all(
                np.diff(
                    pos_630v12r1
                ) == 1
            )
        ):
            per_engine_rows_contiguous_630v12r1 = False

local_reset_630v12r1 = (
    raw_630v12r1.reset_index(
        drop=True
    )
)

reset_preserves_values_630v12r1 = bool(
    tuple(
        local_reset_630v12r1.columns
    )
    ==
    tuple(
        raw_630v12r1.columns
    )
    and
    np.array_equal(
        local_reset_630v12r1.to_numpy(),
        raw_630v12r1.to_numpy(),
        equal_nan=True,
    )
)

if not reset_preserves_values_630v12r1:
    raise RuntimeError(
        "Local RangeIndex projection changed DEV raw values/order."
    )

if not (
    raw_index_unique_630v12r1
    and raw_index_monotonic_630v12r1
    and row_order_is_unit_cycle_sorted_630v12r1
    and per_engine_cycle_sorted_630v12r1
    and per_engine_rows_contiguous_630v12r1
):
    raise RuntimeError(
        "DEV row-order authority is not strong enough to define "
        "a safe positional RangeIndex projection."
    )

FD004_DEV_ORIGINAL_INDEX_630V12R1 = (
    raw_630v12r1.index.copy()
)

FD004_DEV_POSITIONAL_PROJECTION_REQUIRED_630V12R1 = bool(
    not raw_index_is_range_630v12r1
)

FD004_DEV_POSITIONAL_PROJECTION_SAFE_630V12R1 = True

FD004_POSITIONAL_SEMANTICS_RESOLUTION_630V12R1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-12R1",

    "dev_rows":
        int(n_630v12r1),

    "raw_index_is_range_0_to_n_minus_1":
        bool(
            raw_index_is_range_630v12r1
        ),

    "raw_index_unique":
        bool(
            raw_index_unique_630v12r1
        ),

    "raw_index_monotonic_increasing":
        bool(
            raw_index_monotonic_630v12r1
        ),

    "raw_row_order_unit_cycle_sorted":
        bool(
            row_order_is_unit_cycle_sorted_630v12r1
        ),

    "per_engine_cycle_sorted":
        bool(
            per_engine_cycle_sorted_630v12r1
        ),

    "per_engine_rows_contiguous":
        bool(
            per_engine_rows_contiguous_630v12r1
        ),

    "frozen_bridge_preserves_original_index":
        bool(
            all(
                bridge_index_checks_630v12r1.values()
            )
        ),

    "reset_index_drop_preserves_row_values_and_order":
        bool(
            reset_preserves_values_630v12r1
        ),

    "positional_projection_required":
        bool(
            not raw_index_is_range_630v12r1
        ),

    "positional_projection_rule":
        (
            "apply historical positional operator semantics on a local "
            "reset_index(drop=True) copy, then restore the untouched "
            "original FD004 index before downstream preprocessing"
        ),

    "operator_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_POSITIONAL_SEMANTICS_RESOLUTION_SHA256_630V12R1 = (
    hashlib.sha256(
        json.dumps(
            FD004_POSITIONAL_SEMANTICS_RESOLUTION_630V12R1,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_OPERATOR_EXECUTED_630V12R1 = False
FD004_CORRUPTION_EXECUTED_630V12R1 = False
FD004_MODEL_INFERENCE_PERFORMED_630V12R1 = False
FD004_MC_EXECUTION_PERFORMED_630V12R1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V12R1 = False
FD004_THRESHOLD_SELECTED_630V12R1 = False
FD004_POLICY_SELECTED_630V12R1 = False
FD004_TEST_CONTENTS_READ_630V12R1 = False
FD004_RUL_CONTENTS_READ_630V12R1 = False

all_checks_passed_630v12r1 = True

print("=" * 176)

print(
    "STEP 6.30V-12R1 — FD004 ROW-INDEX / POSITIONAL-SEMANTICS "
    "AUTHORITY AUDIT"
)

print("=" * 176)

print("Operator executed                 :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- DEV RAW INDEX AUTHORITY ---")

print(
    "Rows                              :",
    n_630v12r1,
)

print(
    "Index type                        :",
    type(
        raw_630v12r1.index
    ).__name__,
)

print(
    "Index min / max                   :",
    (
        int(
            np.min(
                raw_index_630v12r1
            )
        ),
        int(
            np.max(
                raw_index_630v12r1
            )
        ),
    ),
)

print(
    "Index exact 0..N-1                :",
    raw_index_is_range_630v12r1,
)

print(
    "Index unique                      :",
    raw_index_unique_630v12r1,
)

print(
    "Index monotonic increasing        :",
    raw_index_monotonic_630v12r1,
)

print("\n--- ROW-ORDER AUTHORITY ---")

print(
    "Global unit/cycle order exact     :",
    row_order_is_unit_cycle_sorted_630v12r1,
)

print(
    "Per-engine cycles increasing      :",
    per_engine_cycle_sorted_630v12r1,
)

print(
    "Per-engine rows contiguous        :",
    per_engine_rows_contiguous_630v12r1,
)

print(
    "Reset(drop=True) preserves values :",
    reset_preserves_values_630v12r1,
)

print("\n--- FROZEN BRIDGE INDEX PRESERVATION ---")

for (
    key_630v12r1,
    value_630v12r1,
) in bridge_index_checks_630v12r1.items():

    print(
        f"{key_630v12r1:34s}: "
        f"{value_630v12r1}"
    )

print("\n--- RESOLUTION ---")

print(
    "Positional projection required    :",
    FD004_DEV_POSITIONAL_PROJECTION_REQUIRED_630V12R1,
)

print(
    "Positional projection safe        :",
    FD004_DEV_POSITIONAL_PROJECTION_SAFE_630V12R1,
)

print(
    "Resolution SHA-256                :",
    FD004_POSITIONAL_SEMANTICS_RESOLUTION_SHA256_630V12R1,
)

print(
    "\nPASS — the failure cause is resolved without executing any "
    "corruption operator. FD004 DEV preserves original split-row labels, "
    "while its row order is a safe canonical unit/cycle positional order."
)

print(
    "Next: create an explicitly superseding FD004 operator implementation "
    "that applies the recovered historical positional semantics on a local "
    "RangeIndex projection and restores the original index before returning."
)

In [ ]:
# STEP 6.30V-12R2 — FD004 ROW-IDENTITY / BRIDGE AUTHORITY AUDIT
#
# Audit only.
# NO corruption operator execution.
# NO preprocessing rerun.
# NO model inference.
# NO MC.
# NO DEV target/error use.
# NO threshold/policy.
# NO TEST / official RUL.

import numpy as np
import pandas as pd
import hashlib
import json

required_630v12r2 = (
    "all_checks_passed_630v11",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_NORMALIZED_630F",
    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_SMOKE_NORMALIZED_630V2",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
)

missing_630v12r2 = [
    n for n in required_630v12r2
    if n not in globals()
]

if missing_630v12r2:
    raise RuntimeError(
        "Missing STEP 6.30V-12R2 authority object(s): "
        + ", ".join(missing_630v12r2)
    )

if not bool(all_checks_passed_630v11):
    raise RuntimeError(
        "STEP 6.30V-11 must remain passed."
    )

raw_630v12r2 = FD004_DEV_RAW_630C
norm_630v12r2 = FD004_DEV_NORMALIZED_630F
smoke_raw_630v12r2 = FD004_SMOKE_CORRUPTED_RAW_630V2
smoke_norm_630v12r2 = FD004_SMOKE_NORMALIZED_630V2

frames_630v12r2 = {
    "DEV raw": raw_630v12r2,
    "DEV normalized": norm_630v12r2,
    "Smoke raw": smoke_raw_630v12r2,
    "Smoke normalized": smoke_norm_630v12r2,
}

for label_630v12r2, frame_630v12r2 in frames_630v12r2.items():
    if not isinstance(frame_630v12r2, pd.DataFrame):
        raise RuntimeError(
            f"{label_630v12r2} is not a pandas DataFrame."
        )

identity_cols_630v12r2 = ("unit_id", "cycle")

identity_presence_630v12r2 = {
    label_630v12r2: tuple(
        col_630v12r2
        for col_630v12r2 in identity_cols_630v12r2
        if col_630v12r2 in frame_630v12r2.columns
    )
    for label_630v12r2, frame_630v12r2
    in frames_630v12r2.items()
}

if identity_presence_630v12r2["DEV raw"] != identity_cols_630v12r2:
    raise RuntimeError(
        "DEV raw does not contain both unit_id and cycle."
    )

raw_keys_630v12r2 = (
    raw_630v12r2
    .loc[:, list(identity_cols_630v12r2)]
    .to_numpy()
)

raw_key_tuples_630v12r2 = [
    (int(unit_630v12r2), int(cycle_630v12r2))
    for unit_630v12r2, cycle_630v12r2
    in raw_keys_630v12r2
]

raw_key_unique_630v12r2 = bool(
    len(set(raw_key_tuples_630v12r2))
    == len(raw_key_tuples_630v12r2)
)

if not raw_key_unique_630v12r2:
    raise RuntimeError(
        "DEV raw (unit_id, cycle) identity keys are not unique."
    )

frame_audits_630v12r2 = {}

raw_key_set_630v12r2 = set(
    raw_key_tuples_630v12r2
)

for label_630v12r2, frame_630v12r2 in frames_630v12r2.items():
    has_identity_630v12r2 = bool(
        all(
            col_630v12r2 in frame_630v12r2.columns
            for col_630v12r2 in identity_cols_630v12r2
        )
    )

    audit_630v12r2 = {
        "rows": int(len(frame_630v12r2)),
        "columns": int(frame_630v12r2.shape[1]),
        "index_type": type(frame_630v12r2.index).__name__,
        "index_unique": bool(frame_630v12r2.index.is_unique),
        "index_monotonic_increasing": bool(
            frame_630v12r2.index.is_monotonic_increasing
        ),
        "index_exactly_matches_dev_raw": bool(
            frame_630v12r2.index.equals(
                raw_630v12r2.index
            )
        ),
        "has_unit_id_cycle": has_identity_630v12r2,
        "rowwise_key_match_to_dev_raw": None,
        "same_key_set_as_dev_raw": None,
        "key_unique": None,
    }

    if has_identity_630v12r2:
        frame_keys_630v12r2 = (
            frame_630v12r2
            .loc[:, list(identity_cols_630v12r2)]
            .to_numpy()
        )

        frame_key_tuples_630v12r2 = [
            (int(unit_630v12r2), int(cycle_630v12r2))
            for unit_630v12r2, cycle_630v12r2
            in frame_keys_630v12r2
        ]

        frame_key_set_630v12r2 = set(
            frame_key_tuples_630v12r2
        )

        audit_630v12r2["key_unique"] = bool(
            len(frame_key_set_630v12r2)
            == len(frame_key_tuples_630v12r2)
        )

        audit_630v12r2[
            "same_key_set_as_dev_raw"
        ] = bool(
            frame_key_set_630v12r2
            == raw_key_set_630v12r2
        )

        audit_630v12r2[
            "rowwise_key_match_to_dev_raw"
        ] = bool(
            frame_key_tuples_630v12r2
            == raw_key_tuples_630v12r2
        )

    frame_audits_630v12r2[
        label_630v12r2
    ] = audit_630v12r2

norm_audit_630v12r2 = frame_audits_630v12r2[
    "DEV normalized"
]

if (
    norm_audit_630v12r2["has_unit_id_cycle"]
    and norm_audit_630v12r2["key_unique"]
    and norm_audit_630v12r2["rowwise_key_match_to_dev_raw"]
):
    raw_to_norm_relation_630v12r2 = (
        "EXACT_ROWWISE_UNIT_CYCLE_ALIGNMENT"
    )

elif (
    norm_audit_630v12r2["has_unit_id_cycle"]
    and norm_audit_630v12r2["key_unique"]
    and norm_audit_630v12r2["same_key_set_as_dev_raw"]
):
    raw_to_norm_relation_630v12r2 = (
        "EXACT_UNIT_CYCLE_KEYSET_WITH_PERMUTATION"
    )

elif (
    len(norm_630v12r2) == len(raw_630v12r2)
):
    raw_to_norm_relation_630v12r2 = (
        "ROW_COUNT_ONLY_IDENTITY_NOT_PROVEN"
    )

else:
    raw_to_norm_relation_630v12r2 = (
        "UNRESOLVED"
    )

window_units_630v12r2 = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

window_cycles_630v12r2 = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if (
    window_units_630v12r2.shape
    != window_cycles_630v12r2.shape
):
    raise RuntimeError(
        "Frozen DEV window unit/cycle authority shapes differ."
    )

window_key_tuples_630v12r2 = [
    (int(unit_630v12r2), int(cycle_630v12r2))
    for unit_630v12r2, cycle_630v12r2
    in zip(
        window_units_630v12r2,
        window_cycles_630v12r2,
    )
]

window_keys_unique_630v12r2 = bool(
    len(set(window_key_tuples_630v12r2))
    == len(window_key_tuples_630v12r2)
)

window_keys_all_in_raw_630v12r2 = bool(
    set(window_key_tuples_630v12r2)
    .issubset(raw_key_set_630v12r2)
)

expected_window_keys_630v12r2 = []

for _, grp_630v12r2 in raw_630v12r2.groupby(
    "unit_id",
    sort=False,
):
    grp_sorted_630v12r2 = grp_630v12r2.sort_values(
        "cycle",
        kind="mergesort",
    )

    unit_values_630v12r2 = (
        grp_sorted_630v12r2["unit_id"]
        .to_numpy(dtype=np.int64)
    )

    cycle_values_630v12r2 = (
        grp_sorted_630v12r2["cycle"]
        .to_numpy(dtype=np.int64)
    )

    if len(grp_sorted_630v12r2) < 30:
        continue

    for pos_630v12r2 in range(
        29,
        len(grp_sorted_630v12r2),
    ):
        expected_window_keys_630v12r2.append(
            (
                int(unit_values_630v12r2[pos_630v12r2]),
                int(cycle_values_630v12r2[pos_630v12r2]),
            )
        )

window_endpoint_sequence_exact_630v12r2 = bool(
    expected_window_keys_630v12r2
    == window_key_tuples_630v12r2
)

if (
    raw_to_norm_relation_630v12r2
    == "EXACT_ROWWISE_UNIT_CYCLE_ALIGNMENT"
    and window_endpoint_sequence_exact_630v12r2
):
    proven_identity_authority_630v12r2 = (
        "UNIT_CYCLE_ROWWISE_ORDER"
    )

elif (
    raw_to_norm_relation_630v12r2
    == "EXACT_UNIT_CYCLE_KEYSET_WITH_PERMUTATION"
    and window_keys_all_in_raw_630v12r2
):
    proven_identity_authority_630v12r2 = (
        "UNIT_CYCLE_KEY_MAPPING_REQUIRED"
    )

else:
    proven_identity_authority_630v12r2 = (
        "NOT_YET_PROVEN"
    )

FD004_ROW_IDENTITY_AUDIT_630V12R2 = {
    "dataset": "FD004",
    "stage": "6.30V-12R2",
    "identity_columns": identity_cols_630v12r2,
    "frame_audits": frame_audits_630v12r2,
    "raw_to_normalized_relation":
        raw_to_norm_relation_630v12r2,
    "window_count":
        int(len(window_key_tuples_630v12r2)),
    "window_keys_unique":
        window_keys_unique_630v12r2,
    "window_keys_all_in_raw":
        window_keys_all_in_raw_630v12r2,
    "window_endpoint_sequence_exact":
        window_endpoint_sequence_exact_630v12r2,
    "proven_identity_authority":
        proven_identity_authority_630v12r2,
    "operator_executed": False,
    "corruption_executed": False,
    "preprocessing_rerun": False,
    "model_inference_performed": False,
    "mc_execution_performed": False,
    "dev_targets_errors_used": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2 = (
    hashlib.sha256(
        json.dumps(
            FD004_ROW_IDENTITY_AUDIT_630V12R2,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_PROVEN_ROW_IDENTITY_AUTHORITY_630V12R2 = (
    proven_identity_authority_630v12r2
)

FD004_OPERATOR_EXECUTED_630V12R2 = False
FD004_CORRUPTION_EXECUTED_630V12R2 = False
FD004_PREPROCESSING_RERUN_630V12R2 = False
FD004_MODEL_INFERENCE_PERFORMED_630V12R2 = False
FD004_MC_EXECUTION_PERFORMED_630V12R2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V12R2 = False
FD004_THRESHOLD_SELECTED_630V12R2 = False
FD004_POLICY_SELECTED_630V12R2 = False
FD004_TEST_CONTENTS_READ_630V12R2 = False
FD004_RUL_CONTENTS_READ_630V12R2 = False

all_checks_passed_630v12r2 = True

print("=" * 176)

print(
    "STEP 6.30V-12R2 — FD004 ROW-IDENTITY / "
    "BRIDGE AUTHORITY AUDIT"
)

print("=" * 176)

print("Operator executed                 :", False)
print("Corruption executed               :", False)
print("Preprocessing rerun               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FRAME IDENTITY AUDIT ---")

for label_630v12r2 in (
    "DEV raw",
    "DEV normalized",
    "Smoke raw",
    "Smoke normalized",
):
    audit_630v12r2 = frame_audits_630v12r2[
        label_630v12r2
    ]

    print(f"\n{label_630v12r2}")
    print(
        "  shape                           :",
        (
            audit_630v12r2["rows"],
            audit_630v12r2["columns"],
        ),
    )
    print(
        "  index type                      :",
        audit_630v12r2["index_type"],
    )
    print(
        "  index matches DEV raw           :",
        audit_630v12r2[
            "index_exactly_matches_dev_raw"
        ],
    )
    print(
        "  has unit_id/cycle               :",
        audit_630v12r2[
            "has_unit_id_cycle"
        ],
    )
    print(
        "  key unique                      :",
        audit_630v12r2[
            "key_unique"
        ],
    )
    print(
        "  same key set as DEV raw         :",
        audit_630v12r2[
            "same_key_set_as_dev_raw"
        ],
    )
    print(
        "  rowwise key match to DEV raw    :",
        audit_630v12r2[
            "rowwise_key_match_to_dev_raw"
        ],
    )

print("\n--- RAW -> NORMALIZED RELATION ---")

print(
    "Resolved relation                 :",
    raw_to_norm_relation_630v12r2,
)

print("\n--- WINDOW IDENTITY AUTHORITY ---")

print(
    "Frozen DEV windows                :",
    len(window_key_tuples_630v12r2),
)

print(
    "Window endpoint keys unique       :",
    window_keys_unique_630v12r2,
)

print(
    "All window keys exist in DEV raw  :",
    window_keys_all_in_raw_630v12r2,
)

print(
    "30-cycle endpoint sequence exact  :",
    window_endpoint_sequence_exact_630v12r2,
)

print("\n--- PROVEN AUTHORITY ---")

print(
    "Proven row identity authority     :",
    proven_identity_authority_630v12r2,
)

print(
    "Audit SHA-256                     :",
    FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2,
)

print(
    "\nPASS — this cell only determined how row identity is "
    "actually represented across the frozen FD004 raw, normalized, "
    "smoke, and window authorities. No implementation assumption was "
    "used to execute an experiment."
)

print(
    "Next: adapt the positional corruption bridge only according to "
    "the authority proven above; if identity is not yet proven, stop "
    "and audit further rather than guessing."
)

In [ ]:
# STEP 6.30V-13 — FD004 POSITIONAL-SEMANTICS OPERATOR BRIDGE CORRECTION + SUPERSEDING LOCK

import hashlib
import inspect
import json
from types import MappingProxyType

required_630v13 = (
    "all_checks_passed_630v11",
    "all_checks_passed_630v12r2",
    "FD004_CORRUPTION_OPERATORS_630V11",
    "FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11",
    "FD004_PROVEN_ROW_IDENTITY_AUTHORITY_630V12R2",
    "FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2",
)

missing_630v13 = [
    name_630v13
    for name_630v13 in required_630v13
    if name_630v13 not in globals()
]

if missing_630v13:
    raise RuntimeError(
        "Missing STEP 6.30V-13 authority object(s): "
        + ", ".join(missing_630v13)
    )

if not bool(all_checks_passed_630v11):
    raise RuntimeError(
        "STEP 6.30V-11 must remain passed."
    )

if not bool(all_checks_passed_630v12r2):
    raise RuntimeError(
        "STEP 6.30V-12R2 must pass before STEP 6.30V-13."
    )

EXPECTED_ROW_IDENTITY_AUDIT_SHA256_630V13 = (
    "749a4f8a3b51e5309d1c79375846f751"
    "29425be93ca913ced2ef3441053c75fa"
)

if (
    str(FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2)
    != EXPECTED_ROW_IDENTITY_AUDIT_SHA256_630V13
):
    raise RuntimeError(
        "STEP 6.30V-12R2 row-identity authority fingerprint changed."
    )

if (
    str(FD004_PROVEN_ROW_IDENTITY_AUTHORITY_630V12R2)
    != "UNIT_CYCLE_ROWWISE_ORDER"
):
    raise RuntimeError(
        "Rowwise (unit_id, cycle) positional authority was not proven."
    )

base_ops_630v13 = FD004_CORRUPTION_OPERATORS_630V11

expected_base_families_630v13 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

if tuple(base_ops_630v13.keys()) != expected_base_families_630v13:
    raise RuntimeError(
        "Frozen STEP 6.30V-11 operator registry changed."
    )


FD004_POSITIONAL_BRIDGE_SOURCE_630V13 = r"""
def _fd004_local_positional_projection_630v13(df):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    if "unit_id" not in df.columns or "cycle" not in df.columns:
        raise ValueError(
            "unit_id and cycle are required for the proven row-identity authority."
        )

    original_index = df.index.copy()

    positional = df.reset_index(drop=True).copy(deep=True)

    if len(positional) != len(df):
        raise RuntimeError(
            "Local positional projection changed row count."
        )

    original_keys = (
        df.loc[:, ["unit_id", "cycle"]]
        .to_numpy()
    )

    positional_keys = (
        positional.loc[:, ["unit_id", "cycle"]]
        .to_numpy()
    )

    if not np.array_equal(
        original_keys,
        positional_keys,
    ):
        raise RuntimeError(
            "Local positional projection changed rowwise unit/cycle identity."
        )

    return positional, original_index


def _fd004_restore_original_index_630v13(
    corrupted,
    original_index,
):
    if not isinstance(corrupted, pd.DataFrame):
        raise TypeError(
            "Corruption operator did not return a DataFrame."
        )

    if len(corrupted) != len(original_index):
        raise RuntimeError(
            "Corruption operator changed row count."
        )

    restored = corrupted.copy(deep=True)
    restored.index = original_index.copy()

    return restored


def apply_fd004_bias_630v13(
    df,
    sensor,
    alpha,
    direction
):
    positional, original_index = (
        _fd004_local_positional_projection_630v13(df)
    )

    corrupted, bias_amount = (
        FD004_CORRUPTION_OPERATORS_630V11["bias"](
            positional,
            sensor,
            alpha,
            direction,
        )
    )

    corrupted = _fd004_restore_original_index_630v13(
        corrupted,
        original_index,
    )

    return corrupted, bias_amount


def apply_fd004_noise_630v13(
    df,
    sensor,
    alpha,
    seed=42
):
    positional, original_index = (
        _fd004_local_positional_projection_630v13(df)
    )

    corrupted, noise, target_std = (
        FD004_CORRUPTION_OPERATORS_630V11["noise"](
            positional,
            sensor,
            alpha,
            seed,
        )
    )

    corrupted = _fd004_restore_original_index_630v13(
        corrupted,
        original_index,
    )

    return corrupted, noise, target_std


def apply_fd004_drift_630v13(
    df,
    sensor,
    alpha,
    direction
):
    positional, original_index = (
        _fd004_local_positional_projection_630v13(df)
    )

    corrupted, drift_values, final_drift = (
        FD004_CORRUPTION_OPERATORS_630V11["drift"](
            positional,
            sensor,
            alpha,
            direction,
        )
    )

    corrupted = _fd004_restore_original_index_630v13(
        corrupted,
        original_index,
    )

    return corrupted, drift_values, final_drift


def apply_fd004_dropout_630v13(
    df,
    sensor,
    dropout_rate,
    seed=42
):
    positional, original_index = (
        _fd004_local_positional_projection_630v13(df)
    )

    corrupted, drop_mask, imputation_values = (
        FD004_CORRUPTION_OPERATORS_630V11["dropout"](
            positional,
            sensor,
            dropout_rate,
            seed,
        )
    )

    corrupted = _fd004_restore_original_index_630v13(
        corrupted,
        original_index,
    )

    return corrupted, drop_mask, imputation_values


def apply_fd004_stuck_630v13(
    df,
    sensor,
    affected_fraction
):
    positional, original_index = (
        _fd004_local_positional_projection_630v13(df)
    )

    corrupted, affected_mask, stuck_info = (
        FD004_CORRUPTION_OPERATORS_630V11["stuck"](
            positional,
            sensor,
            affected_fraction,
        )
    )

    corrupted = _fd004_restore_original_index_630v13(
        corrupted,
        original_index,
    )

    return corrupted, affected_mask, stuck_info
"""

compile(
    FD004_POSITIONAL_BRIDGE_SOURCE_630V13,
    "<FD004_POSITIONAL_BRIDGE_SOURCE_630V13>",
    "exec",
)

exec(
    compile(
        FD004_POSITIONAL_BRIDGE_SOURCE_630V13,
        "<FD004_POSITIONAL_BRIDGE_SOURCE_630V13>",
        "exec",
    ),
    globals(),
)

expected_signatures_630v13 = {
    "apply_fd004_bias_630v13":
        "(df, sensor, alpha, direction)",
    "apply_fd004_noise_630v13":
        "(df, sensor, alpha, seed=42)",
    "apply_fd004_drift_630v13":
        "(df, sensor, alpha, direction)",
    "apply_fd004_dropout_630v13":
        "(df, sensor, dropout_rate, seed=42)",
    "apply_fd004_stuck_630v13":
        "(df, sensor, affected_fraction)",
}

observed_signatures_630v13 = {}

for (
    fn_name_630v13,
    expected_signature_630v13,
) in expected_signatures_630v13.items():

    fn_630v13 = globals().get(
        fn_name_630v13
    )

    if fn_630v13 is None or not callable(fn_630v13):
        raise RuntimeError(
            f"Corrected operator missing/not callable: {fn_name_630v13}"
        )

    observed_signature_630v13 = str(
        inspect.signature(fn_630v13)
    )

    observed_signatures_630v13[
        fn_name_630v13
    ] = observed_signature_630v13

    if (
        observed_signature_630v13
        != expected_signature_630v13
    ):
        raise RuntimeError(
            f"Corrected signature mismatch for {fn_name_630v13}: "
            f"{observed_signature_630v13} "
            f"!= {expected_signature_630v13}"
        )


FD004_CORRUPTION_OPERATORS_630V13 = (
    MappingProxyType(
        {
            "bias":
                apply_fd004_bias_630v13,
            "noise":
                apply_fd004_noise_630v13,
            "drift":
                apply_fd004_drift_630v13,
            "dropout":
                apply_fd004_dropout_630v13,
            "stuck":
                apply_fd004_stuck_630v13,
        }
    )
)

FD004_POSITIONAL_BRIDGE_SOURCE_SHA256_630V13 = (
    hashlib.sha256(
        FD004_POSITIONAL_BRIDGE_SOURCE_630V13.encode("utf-8")
    ).hexdigest()
)

FD004_POSITIONAL_OPERATOR_CONTRACT_630V13 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-13",

    "supersedes_implementation_stage":
        "6.30V-11",

    "scientific_corruption_semantics_changed":
        False,

    "historical_operator_semantics_source":
        "FD002_STEP_6.28GC",

    "base_operator_contract_sha256":
        str(
            FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11
        ),

    "row_identity_authority":
        str(
            FD004_PROVEN_ROW_IDENTITY_AUTHORITY_630V12R2
        ),

    "row_identity_audit_sha256":
        str(
            FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2
        ),

    "bridge_rule":
        (
            "reset_index(drop=True) on a local copy; preserve exact "
            "rowwise unit_id/cycle order; execute frozen 6.30V-11 "
            "historical-semantics operator; restore original caller index"
        ),

    "corrected_signatures":
        dict(
            observed_signatures_630v13
        ),

    "operator_source_sha256":
        FD004_POSITIONAL_BRIDGE_SOURCE_SHA256_630V13,

    "operators_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13 = (
    hashlib.sha256(
        json.dumps(
            FD004_POSITIONAL_OPERATOR_CONTRACT_630V13,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_POSITIONAL_OPERATOR_IMPLEMENTATION_LOCKED_630V13 = True
FD004_STEP_630V11_IMPLEMENTATION_SUPERSEDED_BY_630V13 = True

FD004_OPERATOR_EXECUTED_630V13 = False
FD004_CORRUPTION_EXECUTED_630V13 = False
FD004_MODEL_INFERENCE_PERFORMED_630V13 = False
FD004_MC_EXECUTION_PERFORMED_630V13 = False
FD004_DEV_TARGETS_ERRORS_USED_630V13 = False
FD004_THRESHOLD_SELECTED_630V13 = False
FD004_POLICY_SELECTED_630V13 = False
FD004_TEST_CONTENTS_READ_630V13 = False
FD004_RUL_CONTENTS_READ_630V13 = False

all_checks_passed_630v13 = True

print("=" * 176)

print(
    "STEP 6.30V-13 — FD004 POSITIONAL-SEMANTICS OPERATOR "
    "BRIDGE CORRECTION + SUPERSEDING LOCK"
)

print("=" * 176)

print("Corrected operators instantiated  :", True)
print("Operators executed                :", False)
print("Corruption executed               :", False)
print("Scientific semantics changed      :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- PROVEN ROW AUTHORITY ---")

print(
    "Authority                        :",
    FD004_PROVEN_ROW_IDENTITY_AUTHORITY_630V12R2,
)

print(
    "Authority audit SHA-256          :",
    FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2,
)

print(
    "Local positional rule            :",
    "reset_index(drop=True) -> frozen operator -> restore original index",
)

print("\n--- SUPERSEDING OPERATOR SIGNATURES ---")

for family_630v13, fn_630v13 in (
    FD004_CORRUPTION_OPERATORS_630V13.items()
):
    print(
        f"{family_630v13:8s}: "
        f"{fn_630v13.__name__}"
        f"{inspect.signature(fn_630v13)}"
    )

print("\n--- CONTRACT FINGERPRINTS ---")

print(
    "Base 6.30V-11 contract SHA-256   :",
    FD004_FIVE_OPERATOR_CONTRACT_SHA256_630V11,
)

print(
    "Corrected bridge source SHA-256  :",
    FD004_POSITIONAL_BRIDGE_SOURCE_SHA256_630V13,
)

print(
    "Corrected operator contract SHA  :",
    FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13,
)

print(
    "\nPASS — STEP 6.30V-11 is superseded only at the implementation "
    "bridge level. The recovered historical scientific corruption "
    "semantics are unchanged."
)

print(
    "No corruption operator was executed. Next: validate one frozen "
    "admissible scenario per family using the corrected bridge before "
    "the 381-scenario execution."
)

In [ ]:
# STEP 6.30V-14A — FD004 FIVE-FAMILY STRUCTURAL-VALIDATION SCENARIO PRE-FLIGHT LOCK
#
# NO corruption.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.

import numpy as np
import pandas as pd
import hashlib
import json

required_630v14a = (
    "all_checks_passed_630v13",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_SENSOR_SCALE_630V11",
    "FD004_DROPOUT_MEAN_630V11",
    "FD004_SMOKE_SCENARIO_ID_630V2",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
)

missing_630v14a = [
    name_630v14a
    for name_630v14a in required_630v14a
    if name_630v14a not in globals()
]

if missing_630v14a:
    raise RuntimeError(
        "Missing STEP 6.30V-14A authority object(s): "
        + ", ".join(missing_630v14a)
    )

if not bool(all_checks_passed_630v13):
    raise RuntimeError(
        "STEP 6.30V-13 must pass before STEP 6.30V-14A."
    )

manifest_630v14a = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .copy(deep=True)
)

if not isinstance(manifest_630v14a, pd.DataFrame):
    raise RuntimeError(
        "Execution manifest is not a DataFrame."
    )

required_manifest_cols_630v14a = (
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "seed",
)

missing_cols_630v14a = [
    col_630v14a
    for col_630v14a in required_manifest_cols_630v14a
    if col_630v14a not in manifest_630v14a.columns
]

if missing_cols_630v14a:
    raise RuntimeError(
        "Execution manifest is missing required column(s): "
        + ", ".join(missing_cols_630v14a)
    )

expected_families_630v14a = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

observed_families_630v14a = tuple(
    manifest_630v14a[
        "family"
    ].astype(str).drop_duplicates().tolist()
)

if set(observed_families_630v14a) != set(
    expected_families_630v14a
):
    raise RuntimeError(
        "Execution-manifest family set changed."
    )

if tuple(
    FD004_CORRUPTION_OPERATORS_630V13.keys()
) != expected_families_630v14a:
    raise RuntimeError(
        "Corrected operator registry family order/content changed."
    )

sensor_scale_keys_630v14a = set(
    FD004_SENSOR_SCALE_630V11.keys()
)

dropout_keys_630v14a = set(
    FD004_DROPOUT_MEAN_630V11.keys()
)


def _finite_number_630v14a(value):
    if pd.isna(value):
        return False

    try:
        number_630v14a = float(value)
    except Exception:
        return False

    return bool(
        np.isfinite(number_630v14a)
    )


def _row_parameter_status_630v14a(row):
    family_630v14a = str(
        row["family"]
    )

    sensor_630v14a = str(
        row["sensor"]
    )

    status_630v14a = {
        "sensor_known":
            sensor_630v14a
            in sensor_scale_keys_630v14a,

        "alpha_valid":
            None,

        "direction_valid":
            None,

        "fraction_valid":
            None,

        "seed_valid":
            None,

        "dropout_reference_complete":
            None,
    }

    if family_630v14a in (
        "bias",
        "noise",
        "drift",
    ):
        status_630v14a[
            "alpha_valid"
        ] = bool(
            _finite_number_630v14a(
                row["alpha"]
            )
            and float(
                row["alpha"]
            ) > 0.0
        )

    if family_630v14a in (
        "bias",
        "drift",
    ):
        status_630v14a[
            "direction_valid"
        ] = bool(
            _finite_number_630v14a(
                row["direction"]
            )
            and int(
                float(
                    row["direction"]
                )
            ) in (-1, 1)
            and float(
                row["direction"]
            )
            == int(
                float(
                    row["direction"]
                )
            )
        )

    if family_630v14a in (
        "dropout",
        "stuck",
    ):
        status_630v14a[
            "fraction_valid"
        ] = bool(
            _finite_number_630v14a(
                row["requested_fraction"]
            )
            and 0.0
            < float(
                row["requested_fraction"]
            )
            < 1.0
        )

    if family_630v14a in (
        "noise",
        "dropout",
    ):
        status_630v14a[
            "seed_valid"
        ] = bool(
            _finite_number_630v14a(
                row["seed"]
            )
            and float(
                row["seed"]
            )
            == int(
                float(
                    row["seed"]
                )
            )
        )

    if family_630v14a == "dropout":
        status_630v14a[
            "dropout_reference_complete"
        ] = bool(
            all(
                (
                    regime_id_630v14a,
                    sensor_630v14a,
                )
                in dropout_keys_630v14a
                for regime_id_630v14a
                in range(6)
            )
        )

    relevant_checks_630v14a = [
        value_630v14a
        for value_630v14a
        in status_630v14a.values()
        if value_630v14a is not None
    ]

    status_630v14a[
        "all_required_parameters_valid"
    ] = bool(
        all(
            relevant_checks_630v14a
        )
    )

    return status_630v14a


selection_rows_630v14a = {}
selection_status_630v14a = {}

# ------------------------------------------------------------------
# Bias must be the already-frozen smoke authority.
# ------------------------------------------------------------------

bias_candidates_630v14a = (
    manifest_630v14a.loc[
        manifest_630v14a[
            "scenario_id"
        ].astype(str)
        ==
        str(
            FD004_SMOKE_SCENARIO_ID_630V2
        )
    ]
    .copy()
)

if len(bias_candidates_630v14a) != 1:
    raise RuntimeError(
        "Frozen bias smoke scenario does not resolve to exactly "
        "one executable-manifest row."
    )

bias_row_630v14a = (
    bias_candidates_630v14a.iloc[0]
)

if str(
    bias_row_630v14a["family"]
) != "bias":
    raise RuntimeError(
        "Frozen smoke scenario is no longer a bias scenario."
    )

bias_status_630v14a = (
    _row_parameter_status_630v14a(
        bias_row_630v14a
    )
)

if not bias_status_630v14a[
    "all_required_parameters_valid"
]:
    raise RuntimeError(
        "Frozen bias smoke scenario parameters are incomplete."
    )

selection_rows_630v14a[
    "bias"
] = bias_row_630v14a

selection_status_630v14a[
    "bias"
] = bias_status_630v14a


# ------------------------------------------------------------------
# Other four families:
# choose earliest executable scenario whose exact required fields
# are already complete in the frozen execution manifest.
# ------------------------------------------------------------------

for family_630v14a in (
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    family_candidates_630v14a = (
        manifest_630v14a.loc[
            manifest_630v14a[
                "family"
            ].astype(str)
            ==
            family_630v14a
        ]
        .sort_values(
            "scenario_number",
            kind="mergesort",
        )
        .copy()
    )

    if len(family_candidates_630v14a) == 0:
        raise RuntimeError(
            f"No executable scenarios exist for "
            f"family={family_630v14a}."
        )

    selected_row_630v14a = None
    selected_status_630v14a = None

    for _, candidate_row_630v14a in (
        family_candidates_630v14a.iterrows()
    ):
        candidate_status_630v14a = (
            _row_parameter_status_630v14a(
                candidate_row_630v14a
            )
        )

        if candidate_status_630v14a[
            "all_required_parameters_valid"
        ]:
            selected_row_630v14a = (
                candidate_row_630v14a
            )

            selected_status_630v14a = (
                candidate_status_630v14a
            )

            break

    if selected_row_630v14a is None:
        raise RuntimeError(
            f"No structurally complete executable scenario "
            f"was found for family={family_630v14a}."
        )

    selection_rows_630v14a[
        family_630v14a
    ] = selected_row_630v14a

    selection_status_630v14a[
        family_630v14a
    ] = selected_status_630v14a


FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A = {
    family_630v14a: {
        "scenario_id":
            str(
                row_630v14a[
                    "scenario_id"
                ]
            ),

        "scenario_number":
            int(
                row_630v14a[
                    "scenario_number"
                ]
            ),

        "family":
            str(
                row_630v14a[
                    "family"
                ]
            ),

        "sensor":
            str(
                row_630v14a[
                    "sensor"
                ]
            ),

        "severity":
            str(
                row_630v14a[
                    "severity"
                ]
            ),

        "alpha":
            (
                None
                if pd.isna(
                    row_630v14a[
                        "alpha"
                    ]
                )
                else float(
                    row_630v14a[
                        "alpha"
                    ]
                )
            ),

        "direction":
            (
                None
                if pd.isna(
                    row_630v14a[
                        "direction"
                    ]
                )
                else int(
                    float(
                        row_630v14a[
                            "direction"
                        ]
                    )
                )
            ),

        "requested_fraction":
            (
                None
                if pd.isna(
                    row_630v14a[
                        "requested_fraction"
                    ]
                )
                else float(
                    row_630v14a[
                        "requested_fraction"
                    ]
                )
            ),

        "seed":
            (
                None
                if pd.isna(
                    row_630v14a[
                        "seed"
                    ]
                )
                else int(
                    float(
                        row_630v14a[
                            "seed"
                        ]
                    )
                )
            ),
    }
    for (
        family_630v14a,
        row_630v14a,
    )
    in selection_rows_630v14a.items()
}


FD004_STRUCTURAL_VALIDATION_PARAMETER_STATUS_630V14A = {
    family_630v14a:
        dict(
            status_630v14a
        )
    for (
        family_630v14a,
        status_630v14a,
    )
    in selection_status_630v14a.items()
}


scenario_numbers_630v14a = [
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
        family_630v14a
    ][
        "scenario_number"
    ]
    for family_630v14a
    in expected_families_630v14a
]

if len(
    set(
        scenario_numbers_630v14a
    )
) != 5:
    raise RuntimeError(
        "Structural-validation scenarios are not five distinct "
        "manifest scenarios."
    )


FD004_STRUCTURAL_VALIDATION_PREFLIGHT_CONTRACT_630V14A = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14A",

    "operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "scenario_selection_rule":
        (
            "bias=frozen historical smoke authority; "
            "noise/drift/dropout/stuck=earliest executable "
            "scenario with all family-required manifest parameters "
            "already explicit and valid"
        ),

    "selected_scenarios":
        FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A,

    "parameter_status":
        FD004_STRUCTURAL_VALIDATION_PARAMETER_STATUS_630V14A,

    "operator_executed":
        False,

    "corruption_executed":
        False,

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A = (
    hashlib.sha256(
        json.dumps(
            FD004_STRUCTURAL_VALIDATION_PREFLIGHT_CONTRACT_630V14A,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


FD004_STRUCTURAL_VALIDATION_PREFLIGHT_LOCKED_630V14A = True

FD004_OPERATOR_EXECUTED_630V14A = False
FD004_CORRUPTION_EXECUTED_630V14A = False
FD004_PREPROCESSING_RERUN_630V14A = False
FD004_MODEL_INFERENCE_PERFORMED_630V14A = False
FD004_MC_EXECUTION_PERFORMED_630V14A = False
FD004_DEV_TARGETS_ERRORS_USED_630V14A = False
FD004_THRESHOLD_SELECTED_630V14A = False
FD004_POLICY_SELECTED_630V14A = False
FD004_TEST_CONTENTS_READ_630V14A = False
FD004_RUL_CONTENTS_READ_630V14A = False

all_checks_passed_630v14a = True


print("=" * 176)

print(
    "STEP 6.30V-14A — FD004 FIVE-FAMILY STRUCTURAL-VALIDATION "
    "SCENARIO PRE-FLIGHT LOCK"
)

print("=" * 176)

print("Operator executed                 :", False)
print("Corruption executed               :", False)
print("Preprocessing rerun               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- LOCKED VALIDATION SCENARIOS ---")

for family_630v14a in expected_families_630v14a:
    info_630v14a = (
        FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
            family_630v14a
        ]
    )

    print(
        f"\n{family_630v14a.upper()}"
    )

    print(
        "  scenario ID / number            :",
        (
            info_630v14a[
                "scenario_id"
            ],
            info_630v14a[
                "scenario_number"
            ],
        ),
    )

    print(
        "  sensor / severity               :",
        (
            info_630v14a[
                "sensor"
            ],
            info_630v14a[
                "severity"
            ],
        ),
    )

    print(
        "  alpha                           :",
        info_630v14a[
            "alpha"
        ],
    )

    print(
        "  direction                       :",
        info_630v14a[
            "direction"
        ],
    )

    print(
        "  requested fraction              :",
        info_630v14a[
            "requested_fraction"
        ],
    )

    print(
        "  seed                            :",
        info_630v14a[
            "seed"
        ],
    )

    print(
        "  required parameters valid       :",
        FD004_STRUCTURAL_VALIDATION_PARAMETER_STATUS_630V14A[
            family_630v14a
        ][
            "all_required_parameters_valid"
        ],
    )

print("\n--- PREFLIGHT CONTRACT ---")

print(
    "Corrected operator contract SHA   :",
    FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13,
)

print(
    "Preflight contract SHA-256        :",
    FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A,
)

print(
    "\nPASS — five exact admissible structural-validation scenarios "
    "are now selected and frozen from the existing execution manifest "
    "without executing any corruption."
)

print(
    "Next: execute only these five locked scenarios through the corrected "
    "operator bridge and verify their exact family semantics before the "
    "381-scenario experiment."
)

In [ ]:
# STEP 6.30V-14B0 — FD004 FIVE-FAMILY OPERATOR CALL-SEMANTICS AUDIT
#
# NO operator execution.
# NO corruption.
# NO inference.
# NO MC.
# NO DEV outcome use.
# NO threshold/policy.
# NO TEST / RUL.

import inspect
import dis
import hashlib
import json

required_630v14b0 = (
    "all_checks_passed_630v11",
    "all_checks_passed_630v13",
    "all_checks_passed_630v14a",
    "FD004_CORRUPTION_OPERATORS_630V11",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A",
)

missing_630v14b0 = [
    n for n in required_630v14b0
    if n not in globals()
]

if missing_630v14b0:
    raise RuntimeError(
        "Missing STEP 6.30V-14B0 authority object(s): "
        + ", ".join(missing_630v14b0)
    )

if not bool(all_checks_passed_630v11):
    raise RuntimeError("STEP 6.30V-11 must remain passed.")

if not bool(all_checks_passed_630v13):
    raise RuntimeError("STEP 6.30V-13 must remain passed.")

if not bool(all_checks_passed_630v14a):
    raise RuntimeError("STEP 6.30V-14A must remain passed.")

expected_families_630v14b0 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

base_ops_630v14b0 = (
    FD004_CORRUPTION_OPERATORS_630V11
)

bridge_ops_630v14b0 = (
    FD004_CORRUPTION_OPERATORS_630V13
)

if tuple(
    base_ops_630v14b0.keys()
) != expected_families_630v14b0:
    raise RuntimeError(
        "Base operator registry family order/content changed."
    )

if tuple(
    bridge_ops_630v14b0.keys()
) != expected_families_630v14b0:
    raise RuntimeError(
        "Corrected operator registry family order/content changed."
    )


def _code_audit_630v14b0(fn):
    code_obj = getattr(
        fn,
        "__code__",
        None,
    )

    if code_obj is None:
        raise RuntimeError(
            f"Callable {getattr(fn, '__name__', '<unknown>')} "
            "has no Python code object."
        )

    instructions = list(
        dis.get_instructions(fn)
    )

    return_ops = []

    for idx, ins in enumerate(
        instructions
    ):
        if ins.opname == "RETURN_VALUE":
            prev = (
                instructions[idx - 1]
                if idx > 0
                else None
            )

            return_ops.append(
                None
                if prev is None
                else {
                    "opname": prev.opname,
                    "arg": prev.arg,
                    "argrepr": prev.argrepr,
                }
            )

    string_consts = tuple(
        x
        for x in code_obj.co_consts
        if isinstance(x, str)
    )

    return {
        "name":
            fn.__name__,

        "signature":
            str(
                inspect.signature(fn)
            ),

        "argcount":
            int(
                code_obj.co_argcount
            ),

        "posonlyargcount":
            int(
                getattr(
                    code_obj,
                    "co_posonlyargcount",
                    0,
                )
            ),

        "kwonlyargcount":
            int(
                code_obj.co_kwonlyargcount
            ),

        "varnames":
            tuple(
                code_obj.co_varnames[
                    :code_obj.co_nlocals
                ]
            ),

        "global_names":
            tuple(
                code_obj.co_names
            ),

        "string_constants":
            string_consts,

        "return_predecessors":
            tuple(
                None
                if item is None
                else (
                    item["opname"],
                    item["arg"],
                    item["argrepr"],
                )
                for item in return_ops
            ),
    }


base_audits_630v14b0 = {
    family:
        _code_audit_630v14b0(
            base_ops_630v14b0[
                family
            ]
        )
    for family
    in expected_families_630v14b0
}


bridge_audits_630v14b0 = {
    family:
        _code_audit_630v14b0(
            bridge_ops_630v14b0[
                family
            ]
        )
    for family
    in expected_families_630v14b0
}


FD004_OPERATOR_CALL_SEMANTICS_AUDIT_630V14B0 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B0",

    "base_operator_audits":
        base_audits_630v14b0,

    "bridge_operator_audits":
        bridge_audits_630v14b0,

    "selected_validation_scenarios":
        FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A,

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "validation_preflight_sha256":
        str(
            FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A
        ),

    "operator_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_OPERATOR_CALL_SEMANTICS_AUDIT_SHA256_630V14B0 = (
    hashlib.sha256(
        json.dumps(
            FD004_OPERATOR_CALL_SEMANTICS_AUDIT_630V14B0,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_OPERATOR_EXECUTED_630V14B0 = False
FD004_CORRUPTION_EXECUTED_630V14B0 = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B0 = False
FD004_MC_EXECUTION_PERFORMED_630V14B0 = False
FD004_DEV_TARGETS_ERRORS_USED_630V14B0 = False
FD004_THRESHOLD_SELECTED_630V14B0 = False
FD004_POLICY_SELECTED_630V14B0 = False
FD004_TEST_CONTENTS_READ_630V14B0 = False
FD004_RUL_CONTENTS_READ_630V14B0 = False

all_checks_passed_630v14b0 = True


print("=" * 176)

print(
    "STEP 6.30V-14B0 — FD004 FIVE-FAMILY OPERATOR "
    "CALL-SEMANTICS AUDIT"
)

print("=" * 176)

print("Operator executed                 :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\n--- BASE HISTORICAL-SEMANTICS OPERATORS (6.30V-11) ---"
)

for family in expected_families_630v14b0:
    audit = base_audits_630v14b0[
        family
    ]

    print(
        f"\n{family.upper()}"
    )

    print(
        "  callable                        :",
        audit["name"],
    )

    print(
        "  signature                       :",
        audit["signature"],
    )

    print(
        "  local variable names            :",
        audit["varnames"],
    )

    print(
        "  referenced global names         :",
        audit["global_names"],
    )

    print(
        "  string constants                :",
        audit["string_constants"],
    )

    print(
        "  return predecessors             :",
        audit["return_predecessors"],
    )


print(
    "\n--- CORRECTED POSITIONAL BRIDGES (6.30V-13) ---"
)

for family in expected_families_630v14b0:
    audit = bridge_audits_630v14b0[
        family
    ]

    print(
        f"\n{family.upper()}"
    )

    print(
        "  callable                        :",
        audit["name"],
    )

    print(
        "  signature                       :",
        audit["signature"],
    )

    print(
        "  return predecessors             :",
        audit["return_predecessors"],
    )


print(
    "\n--- AUDIT FINGERPRINT ---"
)

print(
    "Operator call-semantics SHA-256   :",
    FD004_OPERATOR_CALL_SEMANTICS_AUDIT_SHA256_630V14B0,
)

print(
    "\nPASS — the exact live callable signatures, referenced names, "
    "string constants, and bytecode return structures were audited "
    "without executing any corruption operator."
)

print(
    "Next: use only these observed call/return semantics to execute "
    "the five already-locked structural validation scenarios."
)

In [ ]:
# STEP 6.30V-14B1 — FD004 BIAS CORRECTED-BRIDGE STRUCTURAL VALIDATION

import numpy as np
import hashlib
import json

required_630v14b1 = (
    "all_checks_passed_630v13",
    "all_checks_passed_630v14a",
    "all_checks_passed_630v14b0",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_SENSOR_SCALE_630V11",
    "FD004_DEV_RAW_630C",
    "FD004_SMOKE_CORRUPTED_RAW_630V2",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
)

missing_630v14b1 = [
    name_630v14b1
    for name_630v14b1 in required_630v14b1
    if name_630v14b1 not in globals()
]

if missing_630v14b1:
    raise RuntimeError(
        "Missing STEP 6.30V-14B1 authority object(s): "
        + ", ".join(missing_630v14b1)
    )

if not (
    bool(all_checks_passed_630v13)
    and bool(all_checks_passed_630v14a)
    and bool(all_checks_passed_630v14b0)
):
    raise RuntimeError(
        "STEP 6.30V-13, 6.30V-14A, and 6.30V-14B0 must remain passed."
    )

scenario_630v14b1 = (
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A["bias"]
)

if (
    str(scenario_630v14b1["scenario_id"]) != "FD004_S007"
    or int(scenario_630v14b1["scenario_number"]) != 7
    or str(scenario_630v14b1["family"]) != "bias"
):
    raise RuntimeError(
        "Locked bias validation scenario changed."
    )

sensor_630v14b1 = str(
    scenario_630v14b1["sensor"]
)

alpha_630v14b1 = float(
    scenario_630v14b1["alpha"]
)

direction_630v14b1 = int(
    scenario_630v14b1["direction"]
)

raw_630v14b1 = FD004_DEV_RAW_630C

raw_snapshot_630v14b1 = (
    raw_630v14b1.copy(deep=True)
)

original_index_630v14b1 = (
    raw_630v14b1.index.copy()
)

result_630v14b1 = (
    FD004_CORRUPTION_OPERATORS_630V13["bias"](
        raw_630v14b1,
        sensor_630v14b1,
        alpha_630v14b1,
        direction_630v14b1,
    )
)

if not (
    isinstance(result_630v14b1, tuple)
    and len(result_630v14b1) == 2
):
    raise RuntimeError(
        "Corrected bias operator return structure changed."
    )

(
    corrupted_630v14b1,
    bias_amount_630v14b1,
) = result_630v14b1

expected_amount_630v14b1 = float(
    direction_630v14b1
    * alpha_630v14b1
    * float(
        FD004_SENSOR_SCALE_630V11[
            sensor_630v14b1
        ]
    )
)

amount_exact_630v14b1 = bool(
    float(bias_amount_630v14b1)
    == expected_amount_630v14b1
)

index_restored_630v14b1 = bool(
    corrupted_630v14b1.index.equals(
        original_index_630v14b1
    )
)

unit_cycle_preserved_630v14b1 = bool(
    np.array_equal(
        corrupted_630v14b1[
            ["unit_id", "cycle"]
        ].to_numpy(),
        raw_630v14b1[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

frozen_smoke_630v14b1 = (
    FD004_SMOKE_CORRUPTED_RAW_630V2
)

columns_exact_630v14b1 = bool(
    tuple(
        corrupted_630v14b1.columns
    )
    ==
    tuple(
        frozen_smoke_630v14b1.columns
    )
)

values_exact_630v14b1 = bool(
    columns_exact_630v14b1
    and
    np.array_equal(
        corrupted_630v14b1.to_numpy(),
        frozen_smoke_630v14b1.to_numpy(),
        equal_nan=True,
    )
)

raw_unchanged_630v14b1 = bool(
    raw_630v14b1.equals(
        raw_snapshot_630v14b1
    )
)

if not all(
    (
        amount_exact_630v14b1,
        index_restored_630v14b1,
        unit_cycle_preserved_630v14b1,
        values_exact_630v14b1,
        raw_unchanged_630v14b1,
    )
):
    raise RuntimeError(
        "Corrected bias bridge failed exact structural validation."
    )

FD004_BIAS_STRUCTURAL_VALIDATION_CONTRACT_630V14B1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B1",

    "scenario_id":
        str(
            scenario_630v14b1["scenario_id"]
        ),

    "scenario_number":
        int(
            scenario_630v14b1["scenario_number"]
        ),

    "sensor":
        sensor_630v14b1,

    "alpha":
        alpha_630v14b1,

    "direction":
        direction_630v14b1,

    "expected_bias_amount":
        expected_amount_630v14b1,

    "returned_bias_amount":
        float(
            bias_amount_630v14b1
        ),

    "bias_amount_exact":
        amount_exact_630v14b1,

    "original_index_restored":
        index_restored_630v14b1,

    "unit_cycle_order_preserved":
        unit_cycle_preserved_630v14b1,

    "frozen_smoke_values_exact":
        values_exact_630v14b1,

    "dev_raw_authority_unchanged":
        raw_unchanged_630v14b1,

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1 = (
    hashlib.sha256(
        json.dumps(
            FD004_BIAS_STRUCTURAL_VALIDATION_CONTRACT_630V14B1,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_BIAS_STRUCTURAL_VALIDATION_LOCKED_630V14B1 = True
FD004_BIAS_OPERATOR_EXECUTED_630V14B1 = True

FD004_FULL_CORRUPTION_EXECUTED_630V14B1 = False
FD004_PREPROCESSING_RERUN_630V14B1 = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B1 = False
FD004_MC_EXECUTION_PERFORMED_630V14B1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V14B1 = False
FD004_THRESHOLD_SELECTED_630V14B1 = False
FD004_POLICY_SELECTED_630V14B1 = False
FD004_TEST_CONTENTS_READ_630V14B1 = False
FD004_RUL_CONTENTS_READ_630V14B1 = False

all_checks_passed_630v14b1 = True

print("=" * 176)

print(
    "STEP 6.30V-14B1 — FD004 BIAS CORRECTED-BRIDGE "
    "STRUCTURAL VALIDATION"
)

print("=" * 176)

print("Bias operator executions          : 1")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- LOCKED SCENARIO ---")

print(
    "Scenario ID / number              :",
    (
        scenario_630v14b1["scenario_id"],
        scenario_630v14b1["scenario_number"],
    ),
)

print(
    "Sensor / alpha / direction        :",
    (
        sensor_630v14b1,
        alpha_630v14b1,
        direction_630v14b1,
    ),
)

print("\n--- EXACT VALIDATION ---")

print(
    "Bias amount exact                 :",
    amount_exact_630v14b1,
)

print(
    "Original index restored           :",
    index_restored_630v14b1,
)

print(
    "unit_id/cycle order preserved     :",
    unit_cycle_preserved_630v14b1,
)

print(
    "Frozen smoke values exact         :",
    values_exact_630v14b1,
)

print(
    "DEV raw authority unchanged       :",
    raw_unchanged_630v14b1,
)

print("\n--- CONTRACT FINGERPRINT ---")

print(
    "Bias structural SHA-256           :",
    FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1,
)

print(
    "\nPASS — corrected FD004 bias execution exactly reproduces "
    "the previously frozen smoke corruption while restoring the "
    "original DEV index and preserving rowwise unit/cycle identity."
)

print(
    "Next: validate the locked noise scenario only."
)

In [ ]:
# STEP 6.30V-14B2 — FD004 NOISE CORRECTED-BRIDGE STRUCTURAL VALIDATION

import numpy as np
import hashlib
import json

required_630v14b2 = (
    "all_checks_passed_630v14b1",
    "all_checks_passed_630v14a",
    "all_checks_passed_630v14b0",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_SENSOR_SCALE_630V11",
    "FD004_DEV_RAW_630C",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1",
)

missing_630v14b2 = [
    name_630v14b2
    for name_630v14b2 in required_630v14b2
    if name_630v14b2 not in globals()
]

if missing_630v14b2:
    raise RuntimeError(
        "Missing STEP 6.30V-14B2 authority object(s): "
        + ", ".join(missing_630v14b2)
    )

if not (
    bool(all_checks_passed_630v14b1)
    and bool(all_checks_passed_630v14a)
    and bool(all_checks_passed_630v14b0)
):
    raise RuntimeError(
        "STEP 6.30V-14B1, 6.30V-14A, and 6.30V-14B0 must remain passed."
    )

scenario_630v14b2 = (
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A["noise"]
)

if (
    str(scenario_630v14b2["scenario_id"]) != "FD004_S130"
    or int(scenario_630v14b2["scenario_number"]) != 130
    or str(scenario_630v14b2["family"]) != "noise"
):
    raise RuntimeError(
        "Locked noise validation scenario changed."
    )

sensor_630v14b2 = str(
    scenario_630v14b2["sensor"]
)

alpha_630v14b2 = float(
    scenario_630v14b2["alpha"]
)

seed_630v14b2 = int(
    scenario_630v14b2["seed"]
)

raw_630v14b2 = FD004_DEV_RAW_630C
raw_snapshot_630v14b2 = raw_630v14b2.copy(deep=True)
original_index_630v14b2 = raw_630v14b2.index.copy()

result_630v14b2 = (
    FD004_CORRUPTION_OPERATORS_630V13["noise"](
        raw_630v14b2,
        sensor_630v14b2,
        alpha_630v14b2,
        seed_630v14b2,
    )
)

if not (
    isinstance(result_630v14b2, tuple)
    and len(result_630v14b2) == 3
):
    raise RuntimeError(
        "Corrected noise operator return structure changed."
    )

(
    corrupted_630v14b2,
    noise_vector_630v14b2,
    target_std_630v14b2,
) = result_630v14b2

expected_target_std_630v14b2 = float(
    alpha_630v14b2
    * float(
        FD004_SENSOR_SCALE_630V11[
            sensor_630v14b2
        ]
    )
)

rng_630v14b2 = np.random.default_rng(
    seed_630v14b2
)

expected_noise_vector_630v14b2 = (
    rng_630v14b2.normal(
        loc=0.0,
        scale=1.0,
        size=len(raw_630v14b2),
    )
    * expected_target_std_630v14b2
)

target_std_exact_630v14b2 = bool(
    float(target_std_630v14b2)
    == expected_target_std_630v14b2
)

noise_vector_exact_630v14b2 = bool(
    np.array_equal(
        np.asarray(
            noise_vector_630v14b2,
            dtype=np.float64,
        ),
        np.asarray(
            expected_noise_vector_630v14b2,
            dtype=np.float64,
        ),
    )
)

expected_sensor_values_630v14b2 = (
    raw_630v14b2[
        sensor_630v14b2
    ].to_numpy(dtype=np.float64)
    + expected_noise_vector_630v14b2
)

corrupted_sensor_exact_630v14b2 = bool(
    np.array_equal(
        corrupted_630v14b2[
            sensor_630v14b2
        ].to_numpy(dtype=np.float64),
        expected_sensor_values_630v14b2,
    )
)

index_restored_630v14b2 = bool(
    corrupted_630v14b2.index.equals(
        original_index_630v14b2
    )
)

unit_cycle_preserved_630v14b2 = bool(
    np.array_equal(
        corrupted_630v14b2[
            ["unit_id", "cycle"]
        ].to_numpy(),
        raw_630v14b2[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

non_target_cols_630v14b2 = [
    col_630v14b2
    for col_630v14b2 in raw_630v14b2.columns
    if col_630v14b2 != sensor_630v14b2
]

non_target_unchanged_630v14b2 = bool(
    np.array_equal(
        corrupted_630v14b2[
            non_target_cols_630v14b2
        ].to_numpy(),
        raw_630v14b2[
            non_target_cols_630v14b2
        ].to_numpy(),
        equal_nan=True,
    )
)

raw_unchanged_630v14b2 = bool(
    raw_630v14b2.equals(
        raw_snapshot_630v14b2
    )
)

noise_all_finite_630v14b2 = bool(
    np.all(
        np.isfinite(
            np.asarray(
                noise_vector_630v14b2,
                dtype=np.float64,
            )
        )
    )
)

corrupted_sensor_all_finite_630v14b2 = bool(
    np.all(
        np.isfinite(
            corrupted_630v14b2[
                sensor_630v14b2
            ].to_numpy(dtype=np.float64)
        )
    )
)

if not all(
    (
        target_std_exact_630v14b2,
        noise_vector_exact_630v14b2,
        corrupted_sensor_exact_630v14b2,
        index_restored_630v14b2,
        unit_cycle_preserved_630v14b2,
        non_target_unchanged_630v14b2,
        raw_unchanged_630v14b2,
        noise_all_finite_630v14b2,
        corrupted_sensor_all_finite_630v14b2,
    )
):
    raise RuntimeError(
        "Corrected noise bridge failed exact structural validation."
    )


def _sha_array_630v14b2(value_630v14b2):
    arr_630v14b2 = np.ascontiguousarray(
        np.asarray(value_630v14b2)
    )

    h_630v14b2 = hashlib.sha256()
    h_630v14b2.update(
        str(arr_630v14b2.dtype).encode("utf-8")
    )
    h_630v14b2.update(
        str(tuple(arr_630v14b2.shape)).encode("utf-8")
    )
    h_630v14b2.update(
        arr_630v14b2.tobytes(order="C")
    )

    return h_630v14b2.hexdigest()


FD004_NOISE_VECTOR_SHA256_630V14B2 = (
    _sha_array_630v14b2(
        np.asarray(
            noise_vector_630v14b2,
            dtype=np.float64,
        )
    )
)

FD004_NOISE_STRUCTURAL_VALIDATION_CONTRACT_630V14B2 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B2",

    "scenario_id":
        str(
            scenario_630v14b2["scenario_id"]
        ),

    "scenario_number":
        int(
            scenario_630v14b2["scenario_number"]
        ),

    "sensor":
        sensor_630v14b2,

    "alpha":
        alpha_630v14b2,

    "seed":
        seed_630v14b2,

    "expected_target_std":
        expected_target_std_630v14b2,

    "returned_target_std":
        float(target_std_630v14b2),

    "target_std_exact":
        target_std_exact_630v14b2,

    "seeded_noise_vector_exact":
        noise_vector_exact_630v14b2,

    "corrupted_sensor_exact":
        corrupted_sensor_exact_630v14b2,

    "original_index_restored":
        index_restored_630v14b2,

    "unit_cycle_order_preserved":
        unit_cycle_preserved_630v14b2,

    "non_target_columns_unchanged":
        non_target_unchanged_630v14b2,

    "dev_raw_authority_unchanged":
        raw_unchanged_630v14b2,

    "noise_vector_all_finite":
        noise_all_finite_630v14b2,

    "corrupted_sensor_all_finite":
        corrupted_sensor_all_finite_630v14b2,

    "noise_vector_sha256":
        FD004_NOISE_VECTOR_SHA256_630V14B2,

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "prior_bias_validation_sha256":
        str(
            FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1
        ),

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2 = (
    hashlib.sha256(
        json.dumps(
            FD004_NOISE_STRUCTURAL_VALIDATION_CONTRACT_630V14B2,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_NOISE_STRUCTURAL_VALIDATION_LOCKED_630V14B2 = True
FD004_NOISE_OPERATOR_EXECUTED_630V14B2 = True

FD004_FULL_CORRUPTION_EXECUTED_630V14B2 = False
FD004_PREPROCESSING_RERUN_630V14B2 = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B2 = False
FD004_MC_EXECUTION_PERFORMED_630V14B2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V14B2 = False
FD004_THRESHOLD_SELECTED_630V14B2 = False
FD004_POLICY_SELECTED_630V14B2 = False
FD004_TEST_CONTENTS_READ_630V14B2 = False
FD004_RUL_CONTENTS_READ_630V14B2 = False

all_checks_passed_630v14b2 = True

print("=" * 176)

print(
    "STEP 6.30V-14B2 — FD004 NOISE CORRECTED-BRIDGE "
    "STRUCTURAL VALIDATION"
)

print("=" * 176)

print("Noise operator executions         : 1")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- LOCKED SCENARIO ---")

print(
    "Scenario ID / number              :",
    (
        scenario_630v14b2["scenario_id"],
        scenario_630v14b2["scenario_number"],
    ),
)

print(
    "Sensor / alpha / seed             :",
    (
        sensor_630v14b2,
        alpha_630v14b2,
        seed_630v14b2,
    ),
)

print("\n--- EXACT VALIDATION ---")

print(
    "Target STD exact                  :",
    target_std_exact_630v14b2,
)

print(
    "Seeded noise vector exact         :",
    noise_vector_exact_630v14b2,
)

print(
    "Corrupted sensor exact            :",
    corrupted_sensor_exact_630v14b2,
)

print(
    "Original index restored           :",
    index_restored_630v14b2,
)

print(
    "unit_id/cycle order preserved     :",
    unit_cycle_preserved_630v14b2,
)

print(
    "Non-target columns unchanged      :",
    non_target_unchanged_630v14b2,
)

print(
    "DEV raw authority unchanged       :",
    raw_unchanged_630v14b2,
)

print(
    "Noise vector finite               :",
    noise_all_finite_630v14b2,
)

print(
    "Corrupted sensor finite           :",
    corrupted_sensor_all_finite_630v14b2,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Noise vector SHA-256              :",
    FD004_NOISE_VECTOR_SHA256_630V14B2,
)

print(
    "Noise structural SHA-256          :",
    FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2,
)

print(
    "\nPASS — corrected FD004 noise execution reproduces the exact "
    "historical seeded Gaussian semantics while preserving the proven "
    "FD004 row-identity authority and leaving all non-target data unchanged."
)

print(
    "Next: validate the locked drift scenario only."
)

In [ ]:
# STEP 6.30V-14B3 — FD004 DRIFT CORRECTED-BRIDGE STRUCTURAL VALIDATION
#
# One locked drift scenario only.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v14b3 = (
    "all_checks_passed_630v14b2",
    "all_checks_passed_630v14a",
    "all_checks_passed_630v14b0",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_SENSOR_SCALE_630V11",
    "FD004_DEV_RAW_630C",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2",
)

missing_630v14b3 = [
    name_630v14b3
    for name_630v14b3 in required_630v14b3
    if name_630v14b3 not in globals()
]

if missing_630v14b3:
    raise RuntimeError(
        "Missing STEP 6.30V-14B3 authority object(s): "
        + ", ".join(missing_630v14b3)
    )

if not (
    bool(all_checks_passed_630v14b2)
    and bool(all_checks_passed_630v14a)
    and bool(all_checks_passed_630v14b0)
):
    raise RuntimeError(
        "Required prior validation stages are not all passed."
    )


# =============================================================================
# 1) LOCKED DRIFT SCENARIO
# =============================================================================

scenario_630v14b3 = (
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
        "drift"
    ]
)

if (
    str(scenario_630v14b3["scenario_id"]) != "FD004_S196"
    or int(scenario_630v14b3["scenario_number"]) != 196
    or str(scenario_630v14b3["family"]) != "drift"
):
    raise RuntimeError(
        "Locked drift validation scenario changed."
    )

sensor_630v14b3 = str(
    scenario_630v14b3["sensor"]
)

alpha_630v14b3 = float(
    scenario_630v14b3["alpha"]
)

direction_630v14b3 = int(
    scenario_630v14b3["direction"]
)

if direction_630v14b3 not in (-1, 1):
    raise RuntimeError(
        "Locked drift direction is invalid."
    )


# =============================================================================
# 2) SOURCE AUTHORITY SNAPSHOT
# =============================================================================

raw_630v14b3 = FD004_DEV_RAW_630C

raw_snapshot_630v14b3 = (
    raw_630v14b3.copy(deep=True)
)

original_index_630v14b3 = (
    raw_630v14b3.index.copy()
)


# =============================================================================
# 3) EXECUTE CORRECTED DRIFT OPERATOR EXACTLY ONCE
# =============================================================================

result_630v14b3 = (
    FD004_CORRUPTION_OPERATORS_630V13["drift"](
        raw_630v14b3,
        sensor_630v14b3,
        alpha_630v14b3,
        direction_630v14b3,
    )
)

if not (
    isinstance(result_630v14b3, tuple)
    and len(result_630v14b3) == 3
):
    raise RuntimeError(
        "Corrected drift operator return structure changed."
    )

(
    corrupted_630v14b3,
    drift_vector_630v14b3,
    final_drift_630v14b3,
) = result_630v14b3


# =============================================================================
# 4) INDEPENDENT EXPECTED DRIFT FROM PROVEN POSITIONAL AUTHORITY
# =============================================================================

expected_final_drift_630v14b3 = float(
    direction_630v14b3
    * alpha_630v14b3
    * float(
        FD004_SENSOR_SCALE_630V11[
            sensor_630v14b3
        ]
    )
)

# Critical:
# Use a LOCAL RangeIndex positional projection because STEP 6.30V-12R2
# proved UNIT_CYCLE_ROWWISE_ORDER as the authority.
positional_raw_630v14b3 = (
    raw_630v14b3
    .reset_index(drop=True)
    .copy(deep=True)
)

expected_drift_vector_630v14b3 = np.zeros(
    len(positional_raw_630v14b3),
    dtype=np.float64,
)

engine_endpoint_checks_630v14b3 = []

for (
    unit_id_630v14b3,
    group_630v14b3,
) in positional_raw_630v14b3.groupby(
    "unit_id",
    sort=False,
):
    group_sorted_630v14b3 = (
        group_630v14b3.sort_values(
            "cycle",
            kind="mergesort",
        )
    )

    positions_630v14b3 = (
        group_sorted_630v14b3.index
        .to_numpy(dtype=np.int64)
    )

    n_rows_630v14b3 = len(
        positions_630v14b3
    )

    if n_rows_630v14b3 < 1:
        raise RuntimeError(
            "Unexpected empty DEV engine trajectory."
        )

    if n_rows_630v14b3 == 1:
        engine_drift_630v14b3 = np.asarray(
            [expected_final_drift_630v14b3],
            dtype=np.float64,
        )
    else:
        engine_drift_630v14b3 = np.linspace(
            0.0,
            expected_final_drift_630v14b3,
            num=n_rows_630v14b3,
            endpoint=True,
            dtype=np.float64,
        )

    expected_drift_vector_630v14b3[
        positions_630v14b3
    ] = engine_drift_630v14b3

    engine_endpoint_checks_630v14b3.append(
        {
            "unit_id":
                int(unit_id_630v14b3),

            "rows":
                int(n_rows_630v14b3),

            "first_drift":
                float(
                    engine_drift_630v14b3[0]
                ),

            "last_drift":
                float(
                    engine_drift_630v14b3[-1]
                ),
        }
    )


# =============================================================================
# 5) EXACT STRUCTURAL CHECKS
# =============================================================================

final_drift_exact_630v14b3 = bool(
    float(final_drift_630v14b3)
    == expected_final_drift_630v14b3
)

drift_vector_exact_630v14b3 = bool(
    np.array_equal(
        np.asarray(
            drift_vector_630v14b3,
            dtype=np.float64,
        ),
        expected_drift_vector_630v14b3,
    )
)

expected_sensor_values_630v14b3 = (
    raw_630v14b3[
        sensor_630v14b3
    ].to_numpy(dtype=np.float64)
    + expected_drift_vector_630v14b3
)

corrupted_sensor_exact_630v14b3 = bool(
    np.array_equal(
        corrupted_630v14b3[
            sensor_630v14b3
        ].to_numpy(dtype=np.float64),
        expected_sensor_values_630v14b3,
    )
)

index_restored_630v14b3 = bool(
    corrupted_630v14b3.index.equals(
        original_index_630v14b3
    )
)

unit_cycle_preserved_630v14b3 = bool(
    np.array_equal(
        corrupted_630v14b3[
            ["unit_id", "cycle"]
        ].to_numpy(),
        raw_630v14b3[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

non_target_cols_630v14b3 = [
    col_630v14b3
    for col_630v14b3 in raw_630v14b3.columns
    if col_630v14b3 != sensor_630v14b3
]

non_target_unchanged_630v14b3 = bool(
    np.array_equal(
        corrupted_630v14b3[
            non_target_cols_630v14b3
        ].to_numpy(),
        raw_630v14b3[
            non_target_cols_630v14b3
        ].to_numpy(),
        equal_nan=True,
    )
)

raw_unchanged_630v14b3 = bool(
    raw_630v14b3.equals(
        raw_snapshot_630v14b3
    )
)

drift_vector_finite_630v14b3 = bool(
    np.all(
        np.isfinite(
            np.asarray(
                drift_vector_630v14b3,
                dtype=np.float64,
            )
        )
    )
)

corrupted_sensor_finite_630v14b3 = bool(
    np.all(
        np.isfinite(
            corrupted_630v14b3[
                sensor_630v14b3
            ].to_numpy(dtype=np.float64)
        )
    )
)

# For multi-row trajectories historical drift must start at 0
# and end at the exact final drift.
engine_endpoints_exact_630v14b3 = True

for info_630v14b3 in engine_endpoint_checks_630v14b3:

    if info_630v14b3["rows"] == 1:
        if (
            info_630v14b3["first_drift"]
            != expected_final_drift_630v14b3
            or
            info_630v14b3["last_drift"]
            != expected_final_drift_630v14b3
        ):
            engine_endpoints_exact_630v14b3 = False
            break

    else:
        if (
            info_630v14b3["first_drift"] != 0.0
            or
            info_630v14b3["last_drift"]
            != expected_final_drift_630v14b3
        ):
            engine_endpoints_exact_630v14b3 = False
            break


if not all(
    (
        final_drift_exact_630v14b3,
        drift_vector_exact_630v14b3,
        corrupted_sensor_exact_630v14b3,
        index_restored_630v14b3,
        unit_cycle_preserved_630v14b3,
        non_target_unchanged_630v14b3,
        raw_unchanged_630v14b3,
        drift_vector_finite_630v14b3,
        corrupted_sensor_finite_630v14b3,
        engine_endpoints_exact_630v14b3,
    )
):
    raise RuntimeError(
        "Corrected drift bridge failed exact structural validation."
    )


# =============================================================================
# 6) EVIDENCE FINGERPRINTS
# =============================================================================

def _sha_array_630v14b3(value_630v14b3):
    arr_630v14b3 = np.ascontiguousarray(
        np.asarray(value_630v14b3)
    )

    h_630v14b3 = hashlib.sha256()

    h_630v14b3.update(
        str(
            arr_630v14b3.dtype
        ).encode("utf-8")
    )

    h_630v14b3.update(
        str(
            tuple(
                arr_630v14b3.shape
            )
        ).encode("utf-8")
    )

    h_630v14b3.update(
        arr_630v14b3.tobytes(order="C")
    )

    return h_630v14b3.hexdigest()


FD004_DRIFT_VECTOR_SHA256_630V14B3 = (
    _sha_array_630v14b3(
        np.asarray(
            drift_vector_630v14b3,
            dtype=np.float64,
        )
    )
)


FD004_DRIFT_STRUCTURAL_VALIDATION_CONTRACT_630V14B3 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B3",

    "scenario_id":
        str(
            scenario_630v14b3[
                "scenario_id"
            ]
        ),

    "scenario_number":
        int(
            scenario_630v14b3[
                "scenario_number"
            ]
        ),

    "sensor":
        sensor_630v14b3,

    "alpha":
        alpha_630v14b3,

    "direction":
        direction_630v14b3,

    "expected_final_drift":
        expected_final_drift_630v14b3,

    "returned_final_drift":
        float(
            final_drift_630v14b3
        ),

    "final_drift_exact":
        final_drift_exact_630v14b3,

    "per_engine_linear_vector_exact":
        drift_vector_exact_630v14b3,

    "per_engine_endpoints_exact":
        engine_endpoints_exact_630v14b3,

    "corrupted_sensor_exact":
        corrupted_sensor_exact_630v14b3,

    "original_index_restored":
        index_restored_630v14b3,

    "unit_cycle_order_preserved":
        unit_cycle_preserved_630v14b3,

    "non_target_columns_unchanged":
        non_target_unchanged_630v14b3,

    "dev_raw_authority_unchanged":
        raw_unchanged_630v14b3,

    "drift_vector_all_finite":
        drift_vector_finite_630v14b3,

    "corrupted_sensor_all_finite":
        corrupted_sensor_finite_630v14b3,

    "drift_vector_sha256":
        FD004_DRIFT_VECTOR_SHA256_630V14B3,

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "prior_noise_validation_sha256":
        str(
            FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2
        ),

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3 = (
    hashlib.sha256(
        json.dumps(
            FD004_DRIFT_STRUCTURAL_VALIDATION_CONTRACT_630V14B3,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) LOCK FLAGS
# =============================================================================

FD004_DRIFT_STRUCTURAL_VALIDATION_LOCKED_630V14B3 = True
FD004_DRIFT_OPERATOR_EXECUTED_630V14B3 = True

FD004_FULL_CORRUPTION_EXECUTED_630V14B3 = False

FD004_PREPROCESSING_RERUN_630V14B3 = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B3 = False
FD004_MC_EXECUTION_PERFORMED_630V14B3 = False

FD004_DEV_TARGETS_ERRORS_USED_630V14B3 = False

FD004_THRESHOLD_SELECTED_630V14B3 = False
FD004_POLICY_SELECTED_630V14B3 = False

FD004_TEST_CONTENTS_READ_630V14B3 = False
FD004_RUL_CONTENTS_READ_630V14B3 = False

all_checks_passed_630v14b3 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-14B3 — FD004 DRIFT CORRECTED-BRIDGE "
    "STRUCTURAL VALIDATION"
)

print("=" * 176)

print("Drift operator executions         : 1")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- LOCKED SCENARIO ---")

print(
    "Scenario ID / number              :",
    (
        scenario_630v14b3[
            "scenario_id"
        ],
        scenario_630v14b3[
            "scenario_number"
        ],
    ),
)

print(
    "Sensor / alpha / direction        :",
    (
        sensor_630v14b3,
        alpha_630v14b3,
        direction_630v14b3,
    ),
)

print("\n--- EXACT VALIDATION ---")

print(
    "Final drift exact                 :",
    final_drift_exact_630v14b3,
)

print(
    "Per-engine ramp vector exact      :",
    drift_vector_exact_630v14b3,
)

print(
    "Per-engine endpoints exact        :",
    engine_endpoints_exact_630v14b3,
)

print(
    "Corrupted sensor exact            :",
    corrupted_sensor_exact_630v14b3,
)

print(
    "Original index restored           :",
    index_restored_630v14b3,
)

print(
    "unit_id/cycle order preserved     :",
    unit_cycle_preserved_630v14b3,
)

print(
    "Non-target columns unchanged      :",
    non_target_unchanged_630v14b3,
)

print(
    "DEV raw authority unchanged       :",
    raw_unchanged_630v14b3,
)

print(
    "Drift vector finite               :",
    drift_vector_finite_630v14b3,
)

print(
    "Corrupted sensor finite           :",
    corrupted_sensor_finite_630v14b3,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Drift vector SHA-256              :",
    FD004_DRIFT_VECTOR_SHA256_630V14B3,
)

print(
    "Drift structural SHA-256          :",
    FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3,
)

print(
    "\nPASS — corrected FD004 drift execution reproduces the exact "
    "historical per-engine linear-ramp semantics on the proven "
    "rowwise unit/cycle positional authority."
)

print(
    "Next: validate the locked dropout scenario only."
)

In [ ]:
# STEP 6.30V-14B4 — FD004 DROPOUT CORRECTED-BRIDGE STRUCTURAL VALIDATION

import numpy as np
import pandas as pd
import hashlib
import json

required_630v14b4 = (
    "all_checks_passed_630v14b3",
    "all_checks_passed_630v14a",
    "all_checks_passed_630v14b0",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_DROPOUT_MEAN_630V11",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3",
)

missing_630v14b4 = [
    name_630v14b4
    for name_630v14b4 in required_630v14b4
    if name_630v14b4 not in globals()
]

if missing_630v14b4:
    raise RuntimeError(
        "Missing STEP 6.30V-14B4 authority object(s): "
        + ", ".join(missing_630v14b4)
    )

if not (
    bool(all_checks_passed_630v14b3)
    and bool(all_checks_passed_630v14a)
    and bool(all_checks_passed_630v14b0)
):
    raise RuntimeError(
        "Required prior validation stages are not all passed."
    )

scenario_630v14b4 = (
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
        "dropout"
    ]
)

if (
    str(scenario_630v14b4["scenario_id"]) != "FD004_S316"
    or int(scenario_630v14b4["scenario_number"]) != 316
    or str(scenario_630v14b4["family"]) != "dropout"
):
    raise RuntimeError(
        "Locked dropout validation scenario changed."
    )

sensor_630v14b4 = str(
    scenario_630v14b4["sensor"]
)

fraction_630v14b4 = float(
    scenario_630v14b4["requested_fraction"]
)

seed_630v14b4 = int(
    scenario_630v14b4["seed"]
)

if not (0.0 < fraction_630v14b4 < 1.0):
    raise RuntimeError(
        "Locked dropout fraction is invalid."
    )

raw_630v14b4 = FD004_DEV_RAW_630C
raw_snapshot_630v14b4 = raw_630v14b4.copy(deep=True)

regime_ids_630v14b4 = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

regime_snapshot_630v14b4 = regime_ids_630v14b4.copy()

if regime_ids_630v14b4.shape != (len(raw_630v14b4),):
    raise RuntimeError(
        "DEV raw/regime-ID alignment changed."
    )

dropout_input_630v14b4 = raw_630v14b4.copy(deep=True)
dropout_input_630v14b4["regime_id"] = regime_ids_630v14b4

dropout_input_snapshot_630v14b4 = (
    dropout_input_630v14b4.copy(deep=True)
)

original_index_630v14b4 = (
    dropout_input_630v14b4.index.copy()
)

result_630v14b4 = (
    FD004_CORRUPTION_OPERATORS_630V13["dropout"](
        dropout_input_630v14b4,
        sensor_630v14b4,
        fraction_630v14b4,
        seed_630v14b4,
    )
)

if not (
    isinstance(result_630v14b4, tuple)
    and len(result_630v14b4) == 3
):
    raise RuntimeError(
        "Corrected dropout operator return structure changed."
    )

(
    corrupted_630v14b4,
    drop_mask_630v14b4,
    imputation_values_630v14b4,
) = result_630v14b4

n_rows_630v14b4 = len(
    dropout_input_630v14b4
)

expected_drop_count_630v14b4 = int(
    round(
        fraction_630v14b4
        * n_rows_630v14b4
    )
)

rng_630v14b4 = np.random.default_rng(
    seed_630v14b4
)

expected_order_630v14b4 = (
    rng_630v14b4.permutation(
        n_rows_630v14b4
    )
)

expected_positions_630v14b4 = (
    expected_order_630v14b4[
        :expected_drop_count_630v14b4
    ]
)

expected_mask_630v14b4 = np.zeros(
    n_rows_630v14b4,
    dtype=bool,
)

expected_mask_630v14b4[
    expected_positions_630v14b4
] = True

returned_mask_630v14b4 = np.asarray(
    drop_mask_630v14b4,
    dtype=bool,
).reshape(-1)

if returned_mask_630v14b4.shape != (
    n_rows_630v14b4,
):
    raise RuntimeError(
        "Returned dropout mask shape changed."
    )

drop_count_exact_630v14b4 = bool(
    int(returned_mask_630v14b4.sum())
    == expected_drop_count_630v14b4
)

drop_mask_exact_630v14b4 = bool(
    np.array_equal(
        returned_mask_630v14b4,
        expected_mask_630v14b4,
    )
)

expected_imputation_630v14b4 = np.empty(
    n_rows_630v14b4,
    dtype=np.float64,
)

for position_630v14b4 in range(
    n_rows_630v14b4
):
    regime_id_630v14b4 = int(
        regime_ids_630v14b4[
            position_630v14b4
        ]
    )

    key_630v14b4 = (
        regime_id_630v14b4,
        sensor_630v14b4,
    )

    if key_630v14b4 not in FD004_DROPOUT_MEAN_630V11:
        raise RuntimeError(
            "Frozen FIT dropout reference missing key: "
            + str(key_630v14b4)
        )

    expected_imputation_630v14b4[
        position_630v14b4
    ] = float(
        FD004_DROPOUT_MEAN_630V11[
            key_630v14b4
        ]
    )

returned_imputation_630v14b4 = np.asarray(
    imputation_values_630v14b4,
    dtype=np.float64,
).reshape(-1)

if returned_imputation_630v14b4.shape != (
    n_rows_630v14b4,
):
    raise RuntimeError(
        "Returned dropout imputation-vector shape changed."
    )

imputation_vector_exact_630v14b4 = bool(
    np.array_equal(
        returned_imputation_630v14b4,
        expected_imputation_630v14b4,
    )
)

expected_sensor_values_630v14b4 = (
    dropout_input_630v14b4[
        sensor_630v14b4
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

expected_sensor_values_630v14b4[
    expected_mask_630v14b4
] = expected_imputation_630v14b4[
    expected_mask_630v14b4
]

corrupted_sensor_exact_630v14b4 = bool(
    np.array_equal(
        corrupted_630v14b4[
            sensor_630v14b4
        ].to_numpy(dtype=np.float64),
        expected_sensor_values_630v14b4,
    )
)

index_restored_630v14b4 = bool(
    corrupted_630v14b4.index.equals(
        original_index_630v14b4
    )
)

unit_cycle_preserved_630v14b4 = bool(
    np.array_equal(
        corrupted_630v14b4[
            ["unit_id", "cycle"]
        ].to_numpy(),
        dropout_input_630v14b4[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

regime_column_preserved_630v14b4 = bool(
    np.array_equal(
        corrupted_630v14b4[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_ids_630v14b4,
    )
)

non_target_cols_630v14b4 = [
    col_630v14b4
    for col_630v14b4 in dropout_input_630v14b4.columns
    if col_630v14b4 != sensor_630v14b4
]

non_target_unchanged_630v14b4 = bool(
    np.array_equal(
        corrupted_630v14b4[
            non_target_cols_630v14b4
        ].to_numpy(),
        dropout_input_630v14b4[
            non_target_cols_630v14b4
        ].to_numpy(),
        equal_nan=True,
    )
)

raw_unchanged_630v14b4 = bool(
    raw_630v14b4.equals(
        raw_snapshot_630v14b4
    )
)

regime_authority_unchanged_630v14b4 = bool(
    np.array_equal(
        regime_ids_630v14b4,
        regime_snapshot_630v14b4,
    )
)

dropout_input_unchanged_630v14b4 = bool(
    dropout_input_630v14b4.equals(
        dropout_input_snapshot_630v14b4
    )
)

corrupted_sensor_finite_630v14b4 = bool(
    np.all(
        np.isfinite(
            corrupted_630v14b4[
                sensor_630v14b4
            ].to_numpy(dtype=np.float64)
        )
    )
)

no_nan_after_imputation_630v14b4 = bool(
    not corrupted_630v14b4[
        sensor_630v14b4
    ].isna().any()
)

if not all(
    (
        drop_count_exact_630v14b4,
        drop_mask_exact_630v14b4,
        imputation_vector_exact_630v14b4,
        corrupted_sensor_exact_630v14b4,
        index_restored_630v14b4,
        unit_cycle_preserved_630v14b4,
        regime_column_preserved_630v14b4,
        non_target_unchanged_630v14b4,
        raw_unchanged_630v14b4,
        regime_authority_unchanged_630v14b4,
        dropout_input_unchanged_630v14b4,
        corrupted_sensor_finite_630v14b4,
        no_nan_after_imputation_630v14b4,
    )
):
    raise RuntimeError(
        "Corrected dropout bridge failed exact structural validation."
    )

def _sha_array_630v14b4(value_630v14b4):
    arr_630v14b4 = np.ascontiguousarray(
        np.asarray(value_630v14b4)
    )

    h_630v14b4 = hashlib.sha256()
    h_630v14b4.update(
        str(arr_630v14b4.dtype).encode("utf-8")
    )
    h_630v14b4.update(
        str(tuple(arr_630v14b4.shape)).encode("utf-8")
    )
    h_630v14b4.update(
        arr_630v14b4.tobytes(order="C")
    )

    return h_630v14b4.hexdigest()

FD004_DROPOUT_MASK_SHA256_630V14B4 = (
    _sha_array_630v14b4(
        returned_mask_630v14b4
    )
)

FD004_DROPOUT_IMPUTATION_VECTOR_SHA256_630V14B4 = (
    _sha_array_630v14b4(
        returned_imputation_630v14b4
    )
)

FD004_DROPOUT_STRUCTURAL_VALIDATION_CONTRACT_630V14B4 = {
    "dataset": "FD004",
    "stage": "6.30V-14B4",
    "scenario_id": str(
        scenario_630v14b4["scenario_id"]
    ),
    "scenario_number": int(
        scenario_630v14b4["scenario_number"]
    ),
    "sensor": sensor_630v14b4,
    "requested_fraction": fraction_630v14b4,
    "seed": seed_630v14b4,
    "row_count": n_rows_630v14b4,
    "expected_drop_count": expected_drop_count_630v14b4,
    "returned_drop_count": int(
        returned_mask_630v14b4.sum()
    ),
    "drop_count_exact": drop_count_exact_630v14b4,
    "seeded_drop_mask_exact": drop_mask_exact_630v14b4,
    "fit_regime_imputation_vector_exact":
        imputation_vector_exact_630v14b4,
    "corrupted_sensor_exact":
        corrupted_sensor_exact_630v14b4,
    "original_index_restored":
        index_restored_630v14b4,
    "unit_cycle_order_preserved":
        unit_cycle_preserved_630v14b4,
    "regime_column_preserved":
        regime_column_preserved_630v14b4,
    "non_target_columns_unchanged":
        non_target_unchanged_630v14b4,
    "dev_raw_authority_unchanged":
        raw_unchanged_630v14b4,
    "dev_regime_authority_unchanged":
        regime_authority_unchanged_630v14b4,
    "local_dropout_input_unchanged":
        dropout_input_unchanged_630v14b4,
    "corrupted_sensor_all_finite":
        corrupted_sensor_finite_630v14b4,
    "no_nan_after_imputation":
        no_nan_after_imputation_630v14b4,
    "drop_mask_sha256":
        FD004_DROPOUT_MASK_SHA256_630V14B4,
    "imputation_vector_sha256":
        FD004_DROPOUT_IMPUTATION_VECTOR_SHA256_630V14B4,
    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),
    "prior_drift_validation_sha256":
        str(
            FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3
        ),
    "preprocessing_rerun": False,
    "model_inference_performed": False,
    "mc_execution_performed": False,
    "dev_targets_errors_used": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4 = (
    hashlib.sha256(
        json.dumps(
            FD004_DROPOUT_STRUCTURAL_VALIDATION_CONTRACT_630V14B4,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_DROPOUT_STRUCTURAL_VALIDATION_LOCKED_630V14B4 = True
FD004_DROPOUT_OPERATOR_EXECUTED_630V14B4 = True
FD004_FULL_CORRUPTION_EXECUTED_630V14B4 = False
FD004_PREPROCESSING_RERUN_630V14B4 = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B4 = False
FD004_MC_EXECUTION_PERFORMED_630V14B4 = False
FD004_DEV_TARGETS_ERRORS_USED_630V14B4 = False
FD004_THRESHOLD_SELECTED_630V14B4 = False
FD004_POLICY_SELECTED_630V14B4 = False
FD004_TEST_CONTENTS_READ_630V14B4 = False
FD004_RUL_CONTENTS_READ_630V14B4 = False

all_checks_passed_630v14b4 = True

print("=" * 176)
print(
    "STEP 6.30V-14B4 — FD004 DROPOUT CORRECTED-BRIDGE "
    "STRUCTURAL VALIDATION"
)
print("=" * 176)

print("Dropout operator executions       : 1")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- LOCKED SCENARIO ---")
print(
    "Scenario ID / number              :",
    (
        scenario_630v14b4["scenario_id"],
        scenario_630v14b4["scenario_number"],
    ),
)
print(
    "Sensor / fraction / seed          :",
    (
        sensor_630v14b4,
        fraction_630v14b4,
        seed_630v14b4,
    ),
)

print("\n--- EXACT VALIDATION ---")
print(
    "Expected / returned drop count    :",
    (
        expected_drop_count_630v14b4,
        int(returned_mask_630v14b4.sum()),
    ),
)
print("Drop count exact                  :", drop_count_exact_630v14b4)
print("Seeded drop mask exact            :", drop_mask_exact_630v14b4)
print("FIT regime imputation exact       :", imputation_vector_exact_630v14b4)
print("Corrupted sensor exact            :", corrupted_sensor_exact_630v14b4)
print("Original index restored           :", index_restored_630v14b4)
print("unit_id/cycle order preserved     :", unit_cycle_preserved_630v14b4)
print("regime_id column preserved        :", regime_column_preserved_630v14b4)
print("Non-target columns unchanged      :", non_target_unchanged_630v14b4)
print("DEV raw authority unchanged       :", raw_unchanged_630v14b4)
print("DEV regime authority unchanged    :", regime_authority_unchanged_630v14b4)
print("Local dropout input unchanged     :", dropout_input_unchanged_630v14b4)
print("Corrupted sensor finite           :", corrupted_sensor_finite_630v14b4)
print("No NaN after FIT imputation       :", no_nan_after_imputation_630v14b4)

print("\n--- EVIDENCE FINGERPRINTS ---")
print(
    "Drop mask SHA-256                 :",
    FD004_DROPOUT_MASK_SHA256_630V14B4,
)
print(
    "Imputation vector SHA-256         :",
    FD004_DROPOUT_IMPUTATION_VECTOR_SHA256_630V14B4,
)
print(
    "Dropout structural SHA-256        :",
    FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4,
)

print(
    "\nPASS — corrected FD004 dropout execution reproduces the exact "
    "historical seeded exact-count masking and frozen FIT-only "
    "regime-specific imputation semantics."
)

print(
    "Next: validate the locked stuck scenario only."
)

In [ ]:
# STEP 6.30V-14B5A — FD004 STUCK CORRECTED-BRIDGE
# CORE STRUCTURAL VALIDATION + INFO-SCHEMA CAPTURE

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v14b5a = (
    "all_checks_passed_630v14b4",
    "all_checks_passed_630v14a",
    "all_checks_passed_630v14b0",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_DEV_RAW_630C",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4",
)

missing_630v14b5a = [
    name_630v14b5a
    for name_630v14b5a in required_630v14b5a
    if name_630v14b5a not in globals()
]

if missing_630v14b5a:
    raise RuntimeError(
        "Missing STEP 6.30V-14B5A authority object(s): "
        + ", ".join(missing_630v14b5a)
    )

if not (
    bool(all_checks_passed_630v14b4)
    and bool(all_checks_passed_630v14a)
    and bool(all_checks_passed_630v14b0)
):
    raise RuntimeError(
        "Required prior validation stages are not all passed."
    )


# =============================================================================
# 1) LOCKED STUCK SCENARIO
# =============================================================================

scenario_630v14b5a = (
    FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
        "stuck"
    ]
)

if (
    str(scenario_630v14b5a["scenario_id"]) != "FD004_S379"
    or int(scenario_630v14b5a["scenario_number"]) != 379
    or str(scenario_630v14b5a["family"]) != "stuck"
):
    raise RuntimeError(
        "Locked stuck validation scenario changed."
    )

sensor_630v14b5a = str(
    scenario_630v14b5a["sensor"]
)

fraction_630v14b5a = float(
    scenario_630v14b5a["requested_fraction"]
)

if not (0.0 < fraction_630v14b5a < 1.0):
    raise RuntimeError(
        "Locked stuck affected fraction is invalid."
    )


# =============================================================================
# 2) SOURCE AUTHORITY SNAPSHOT
# =============================================================================

raw_630v14b5a = FD004_DEV_RAW_630C

raw_snapshot_630v14b5a = (
    raw_630v14b5a.copy(deep=True)
)

original_index_630v14b5a = (
    raw_630v14b5a.index.copy()
)


# =============================================================================
# 3) EXECUTE CORRECTED STUCK OPERATOR EXACTLY ONCE
# =============================================================================

result_630v14b5a = (
    FD004_CORRUPTION_OPERATORS_630V13["stuck"](
        raw_630v14b5a,
        sensor_630v14b5a,
        fraction_630v14b5a,
    )
)

if not (
    isinstance(result_630v14b5a, tuple)
    and len(result_630v14b5a) == 3
):
    raise RuntimeError(
        "Corrected stuck operator return structure changed."
    )

(
    corrupted_630v14b5a,
    affected_mask_630v14b5a,
    stuck_info_630v14b5a,
) = result_630v14b5a

if not isinstance(
    stuck_info_630v14b5a,
    pd.DataFrame,
):
    raise RuntimeError(
        "Returned stuck_info is not a pandas DataFrame."
    )


# =============================================================================
# 4) INDEPENDENT HISTORICAL STUCK SEMANTICS
#    Using the proven local positional RangeIndex authority.
# =============================================================================

positional_raw_630v14b5a = (
    raw_630v14b5a
    .reset_index(drop=True)
    .copy(deep=True)
)

n_rows_total_630v14b5a = len(
    positional_raw_630v14b5a
)

expected_mask_630v14b5a = np.zeros(
    n_rows_total_630v14b5a,
    dtype=bool,
)

expected_sensor_630v14b5a = (
    positional_raw_630v14b5a[
        sensor_630v14b5a
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

expected_records_630v14b5a = []

for (
    unit_id_630v14b5a,
    group_630v14b5a,
) in positional_raw_630v14b5a.groupby(
    "unit_id",
    sort=False,
):

    group_sorted_630v14b5a = (
        group_630v14b5a.sort_values(
            "cycle",
            kind="mergesort",
        )
    )

    positions_630v14b5a = (
        group_sorted_630v14b5a.index
        .to_numpy(dtype=np.int64)
    )

    cycles_630v14b5a = (
        group_sorted_630v14b5a[
            "cycle"
        ]
        .to_numpy(dtype=np.int64)
    )

    n_engine_rows_630v14b5a = int(
        len(positions_630v14b5a)
    )

    if n_engine_rows_630v14b5a < 2:
        raise RuntimeError(
            "Historical stuck semantics require at least "
            "two cycles per engine."
        )

    n_affected_630v14b5a = int(
        round(
            fraction_630v14b5a
            * n_engine_rows_630v14b5a
        )
    )

    n_affected_630v14b5a = max(
        1,
        min(
            n_affected_630v14b5a,
            n_engine_rows_630v14b5a - 1,
        ),
    )

    onset_position_630v14b5a = int(
        n_engine_rows_630v14b5a
        - n_affected_630v14b5a
    )

    source_position_630v14b5a = int(
        positions_630v14b5a[
            onset_position_630v14b5a - 1
        ]
    )

    affected_positions_630v14b5a = (
        positions_630v14b5a[
            onset_position_630v14b5a:
        ]
    )

    freeze_value_630v14b5a = float(
        positional_raw_630v14b5a.loc[
            source_position_630v14b5a,
            sensor_630v14b5a,
        ]
    )

    expected_mask_630v14b5a[
        affected_positions_630v14b5a
    ] = True

    expected_sensor_630v14b5a[
        affected_positions_630v14b5a
    ] = freeze_value_630v14b5a

    expected_records_630v14b5a.append(
        {
            "unit_id":
                int(unit_id_630v14b5a),

            "trajectory_length":
                int(n_engine_rows_630v14b5a),

            "requested_fraction":
                float(fraction_630v14b5a),

            "affected_count":
                int(n_affected_630v14b5a),

            "actual_fraction":
                float(
                    n_affected_630v14b5a
                    / n_engine_rows_630v14b5a
                ),

            "source_cycle":
                int(
                    cycles_630v14b5a[
                        onset_position_630v14b5a - 1
                    ]
                ),

            "stuck_start_cycle":
                int(
                    cycles_630v14b5a[
                        onset_position_630v14b5a
                    ]
                ),

            "freeze_value":
                float(
                    freeze_value_630v14b5a
                ),
        }
    )


# =============================================================================
# 5) CORE EXACT CHECKS — NO ASSUMPTION ABOUT stuck_info COLUMN NAMES
# =============================================================================

returned_mask_630v14b5a = np.asarray(
    affected_mask_630v14b5a,
    dtype=bool,
).reshape(-1)

if returned_mask_630v14b5a.shape != (
    n_rows_total_630v14b5a,
):
    raise RuntimeError(
        "Returned stuck affected-mask shape changed."
    )

mask_exact_630v14b5a = bool(
    np.array_equal(
        returned_mask_630v14b5a,
        expected_mask_630v14b5a,
    )
)

affected_count_exact_630v14b5a = bool(
    int(
        returned_mask_630v14b5a.sum()
    )
    ==
    int(
        expected_mask_630v14b5a.sum()
    )
)

corrupted_sensor_exact_630v14b5a = bool(
    np.array_equal(
        corrupted_630v14b5a[
            sensor_630v14b5a
        ].to_numpy(dtype=np.float64),
        expected_sensor_630v14b5a,
    )
)

index_restored_630v14b5a = bool(
    corrupted_630v14b5a.index.equals(
        original_index_630v14b5a
    )
)

unit_cycle_preserved_630v14b5a = bool(
    np.array_equal(
        corrupted_630v14b5a[
            ["unit_id", "cycle"]
        ].to_numpy(),
        raw_630v14b5a[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

non_target_cols_630v14b5a = [
    col_630v14b5a
    for col_630v14b5a
    in raw_630v14b5a.columns
    if col_630v14b5a
    != sensor_630v14b5a
]

non_target_unchanged_630v14b5a = bool(
    np.array_equal(
        corrupted_630v14b5a[
            non_target_cols_630v14b5a
        ].to_numpy(),
        raw_630v14b5a[
            non_target_cols_630v14b5a
        ].to_numpy(),
        equal_nan=True,
    )
)

raw_unchanged_630v14b5a = bool(
    raw_630v14b5a.equals(
        raw_snapshot_630v14b5a
    )
)

sensor_finite_630v14b5a = bool(
    np.all(
        np.isfinite(
            corrupted_630v14b5a[
                sensor_630v14b5a
            ].to_numpy(dtype=np.float64)
        )
    )
)

engine_count_630v14b5a = int(
    positional_raw_630v14b5a[
        "unit_id"
    ].nunique()
)

stuck_info_row_count_exact_630v14b5a = bool(
    len(stuck_info_630v14b5a)
    == engine_count_630v14b5a
)

stuck_info_index_unique_630v14b5a = bool(
    stuck_info_630v14b5a.index.is_unique
)

if not all(
    (
        mask_exact_630v14b5a,
        affected_count_exact_630v14b5a,
        corrupted_sensor_exact_630v14b5a,
        index_restored_630v14b5a,
        unit_cycle_preserved_630v14b5a,
        non_target_unchanged_630v14b5a,
        raw_unchanged_630v14b5a,
        sensor_finite_630v14b5a,
        stuck_info_row_count_exact_630v14b5a,
        stuck_info_index_unique_630v14b5a,
    )
):
    raise RuntimeError(
        "Corrected stuck bridge failed core structural validation."
    )


# =============================================================================
# 6) RETAIN INDEPENDENT EXPECTED RECORDS FOR B5B
# =============================================================================

FD004_STUCK_EXPECTED_RECORDS_630V14B5A = (
    pd.DataFrame(
        expected_records_630v14b5a
    )
)

FD004_STUCK_RETURNED_INFO_630V14B5A = (
    stuck_info_630v14b5a.copy(deep=True)
)

FD004_STUCK_RETURNED_MASK_630V14B5A = (
    returned_mask_630v14b5a.copy()
)


def _sha_array_630v14b5a(value_630v14b5a):
    arr_630v14b5a = np.ascontiguousarray(
        np.asarray(value_630v14b5a)
    )

    h_630v14b5a = hashlib.sha256()

    h_630v14b5a.update(
        str(arr_630v14b5a.dtype).encode("utf-8")
    )

    h_630v14b5a.update(
        str(
            tuple(
                arr_630v14b5a.shape
            )
        ).encode("utf-8")
    )

    h_630v14b5a.update(
        arr_630v14b5a.tobytes(order="C")
    )

    return h_630v14b5a.hexdigest()


FD004_STUCK_MASK_SHA256_630V14B5A = (
    _sha_array_630v14b5a(
        returned_mask_630v14b5a
    )
)


# =============================================================================
# 7) CORE CONTRACT
# =============================================================================

FD004_STUCK_CORE_VALIDATION_CONTRACT_630V14B5A = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B5A",

    "scenario_id":
        str(
            scenario_630v14b5a[
                "scenario_id"
            ]
        ),

    "scenario_number":
        int(
            scenario_630v14b5a[
                "scenario_number"
            ]
        ),

    "sensor":
        sensor_630v14b5a,

    "requested_fraction":
        fraction_630v14b5a,

    "engine_count":
        engine_count_630v14b5a,

    "affected_count":
        int(
            returned_mask_630v14b5a.sum()
        ),

    "affected_mask_exact":
        mask_exact_630v14b5a,

    "affected_count_exact":
        affected_count_exact_630v14b5a,

    "corrupted_sensor_exact":
        corrupted_sensor_exact_630v14b5a,

    "original_index_restored":
        index_restored_630v14b5a,

    "unit_cycle_order_preserved":
        unit_cycle_preserved_630v14b5a,

    "non_target_columns_unchanged":
        non_target_unchanged_630v14b5a,

    "dev_raw_authority_unchanged":
        raw_unchanged_630v14b5a,

    "corrupted_sensor_all_finite":
        sensor_finite_630v14b5a,

    "stuck_info_is_dataframe":
        True,

    "stuck_info_row_count_exact":
        stuck_info_row_count_exact_630v14b5a,

    "stuck_info_columns_observed":
        tuple(
            str(c)
            for c
            in stuck_info_630v14b5a.columns
        ),

    "stuck_info_exact_semantic_validation":
        "DEFERRED_TO_6.30V-14B5B_AFTER_SCHEMA_OBSERVATION",

    "mask_sha256":
        FD004_STUCK_MASK_SHA256_630V14B5A,

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "prior_dropout_validation_sha256":
        str(
            FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4
        ),

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_STUCK_CORE_VALIDATION_SHA256_630V14B5A = (
    hashlib.sha256(
        json.dumps(
            FD004_STUCK_CORE_VALIDATION_CONTRACT_630V14B5A,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_STUCK_OPERATOR_EXECUTED_630V14B5A = True
FD004_STUCK_CORE_VALIDATION_LOCKED_630V14B5A = True

FD004_FULL_CORRUPTION_EXECUTED_630V14B5A = False
FD004_PREPROCESSING_RERUN_630V14B5A = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B5A = False
FD004_MC_EXECUTION_PERFORMED_630V14B5A = False
FD004_DEV_TARGETS_ERRORS_USED_630V14B5A = False
FD004_THRESHOLD_SELECTED_630V14B5A = False
FD004_POLICY_SELECTED_630V14B5A = False
FD004_TEST_CONTENTS_READ_630V14B5A = False
FD004_RUL_CONTENTS_READ_630V14B5A = False

all_checks_passed_630v14b5a = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-14B5A — FD004 STUCK CORRECTED-BRIDGE "
    "CORE STRUCTURAL VALIDATION + INFO-SCHEMA CAPTURE"
)

print("=" * 176)

print("Stuck operator executions         : 1")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- LOCKED SCENARIO ---")

print(
    "Scenario ID / number              :",
    (
        scenario_630v14b5a[
            "scenario_id"
        ],
        scenario_630v14b5a[
            "scenario_number"
        ],
    ),
)

print(
    "Sensor / affected fraction        :",
    (
        sensor_630v14b5a,
        fraction_630v14b5a,
    ),
)

print("\n--- CORE EXACT VALIDATION ---")

print(
    "Affected mask exact               :",
    mask_exact_630v14b5a,
)

print(
    "Affected count exact              :",
    affected_count_exact_630v14b5a,
)

print(
    "Returned affected rows            :",
    int(
        returned_mask_630v14b5a.sum()
    ),
)

print(
    "Corrupted sensor exact            :",
    corrupted_sensor_exact_630v14b5a,
)

print(
    "Original index restored           :",
    index_restored_630v14b5a,
)

print(
    "unit_id/cycle order preserved     :",
    unit_cycle_preserved_630v14b5a,
)

print(
    "Non-target columns unchanged      :",
    non_target_unchanged_630v14b5a,
)

print(
    "DEV raw authority unchanged       :",
    raw_unchanged_630v14b5a,
)

print(
    "Corrupted sensor finite           :",
    sensor_finite_630v14b5a,
)

print("\n--- RETURNED stuck_info SCHEMA (OBSERVED, NOT ASSUMED) ---")

print(
    "stuck_info type                   :",
    type(
        stuck_info_630v14b5a
    ).__name__,
)

print(
    "stuck_info shape                  :",
    stuck_info_630v14b5a.shape,
)

print(
    "Expected engine rows              :",
    engine_count_630v14b5a,
)

print(
    "stuck_info row count exact        :",
    stuck_info_row_count_exact_630v14b5a,
)

print(
    "stuck_info columns                :",
    tuple(
        stuck_info_630v14b5a.columns
    ),
)

print(
    "stuck_info dtypes                 :",
    tuple(
        str(dtype_630v14b5a)
        for dtype_630v14b5a
        in stuck_info_630v14b5a.dtypes
    ),
)

print("\nFirst returned stuck_info row:")
print(
    stuck_info_630v14b5a.head(1).to_dict(
        orient="records"
    )
)

print("\nFirst independently-derived expected record:")
print(
    FD004_STUCK_EXPECTED_RECORDS_630V14B5A
    .head(1)
    .to_dict(orient="records")
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Stuck mask SHA-256                :",
    FD004_STUCK_MASK_SHA256_630V14B5A,
)

print(
    "Stuck core validation SHA-256     :",
    FD004_STUCK_CORE_VALIDATION_SHA256_630V14B5A,
)

print(
    "\nPASS — the actual stuck corruption and affected mask exactly "
    "match the independently reconstructed historical terminal-freeze "
    "semantics."
)

print(
    "The returned stuck_info schema has only been observed here, not "
    "assumed. Next: compare that resident stuck_info against the already-"
    "derived expected engine records without rerunning the operator."
)

In [ ]:
# STEP 6.30V-14B5B — FD004 STUCK INFO SEMANTIC VALIDATION + FINAL STUCK LOCK
#
# Uses only resident outputs from 6.30V-14B5A.
# NO corruption/operator rerun.
# NO preprocessing.
# NO inference.
# NO MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v14b5b = (
    "all_checks_passed_630v14b5a",
    "FD004_STUCK_EXPECTED_RECORDS_630V14B5A",
    "FD004_STUCK_RETURNED_INFO_630V14B5A",
    "FD004_STUCK_MASK_SHA256_630V14B5A",
    "FD004_STUCK_CORE_VALIDATION_SHA256_630V14B5A",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
)

missing_630v14b5b = [
    name_630v14b5b
    for name_630v14b5b in required_630v14b5b
    if name_630v14b5b not in globals()
]

if missing_630v14b5b:
    raise RuntimeError(
        "Missing STEP 6.30V-14B5B authority object(s): "
        + ", ".join(missing_630v14b5b)
    )

if not bool(all_checks_passed_630v14b5a):
    raise RuntimeError(
        "STEP 6.30V-14B5A must remain passed."
    )


# =============================================================================
# 1) RESIDENT AUTHORITIES — NO OPERATOR CALL
# =============================================================================

returned_630v14b5b = (
    FD004_STUCK_RETURNED_INFO_630V14B5A
    .copy(deep=True)
)

expected_630v14b5b = (
    FD004_STUCK_EXPECTED_RECORDS_630V14B5A
    .copy(deep=True)
)

if not isinstance(returned_630v14b5b, pd.DataFrame):
    raise RuntimeError(
        "Resident returned stuck_info is not a DataFrame."
    )

if not isinstance(expected_630v14b5b, pd.DataFrame):
    raise RuntimeError(
        "Resident expected stuck records are not a DataFrame."
    )


# =============================================================================
# 2) OBSERVED SCHEMA AUTHORITY
# =============================================================================

observed_schema_630v14b5b = (
    "unit_id",
    "trajectory_length",
    "requested_fraction",
    "affected_count",
    "actual_fraction",
    "source_cycle",
    "stuck_start_cycle",
    "freeze_value",
)

returned_schema_exact_630v14b5b = bool(
    tuple(returned_630v14b5b.columns)
    == observed_schema_630v14b5b
)

expected_schema_exact_630v14b5b = bool(
    tuple(expected_630v14b5b.columns)
    == observed_schema_630v14b5b
)

row_count_exact_630v14b5b = bool(
    len(returned_630v14b5b)
    == len(expected_630v14b5b)
    == 50
)

if not all(
    (
        returned_schema_exact_630v14b5b,
        expected_schema_exact_630v14b5b,
        row_count_exact_630v14b5b,
    )
):
    raise RuntimeError(
        "Resident stuck_info schema/row authority changed."
    )


# =============================================================================
# 3) EXACT ROW ORDER
# =============================================================================

returned_unit_ids_630v14b5b = (
    returned_630v14b5b["unit_id"]
    .to_numpy()
)

expected_unit_ids_630v14b5b = (
    expected_630v14b5b["unit_id"]
    .to_numpy()
)

unit_id_order_exact_630v14b5b = bool(
    np.array_equal(
        returned_unit_ids_630v14b5b.astype(np.int64),
        expected_unit_ids_630v14b5b.astype(np.int64),
    )
)

if not unit_id_order_exact_630v14b5b:
    raise RuntimeError(
        "Returned stuck_info engine row order differs from "
        "the independently reconstructed authority."
    )


# =============================================================================
# 4) COLUMN-WISE SEMANTIC EQUALITY
# =============================================================================

integer_semantic_cols_630v14b5b = (
    "unit_id",
    "trajectory_length",
    "affected_count",
    "source_cycle",
    "stuck_start_cycle",
)

float_semantic_cols_630v14b5b = (
    "requested_fraction",
    "actual_fraction",
    "freeze_value",
)

integer_column_checks_630v14b5b = {}
float_column_checks_630v14b5b = {}


for col_630v14b5b in integer_semantic_cols_630v14b5b:

    returned_values_630v14b5b = (
        returned_630v14b5b[col_630v14b5b]
        .to_numpy(dtype=np.float64)
    )

    expected_values_630v14b5b = (
        expected_630v14b5b[col_630v14b5b]
        .to_numpy(dtype=np.float64)
    )

    finite_630v14b5b = bool(
        np.all(
            np.isfinite(
                returned_values_630v14b5b
            )
        )
        and
        np.all(
            np.isfinite(
                expected_values_630v14b5b
            )
        )
    )

    returned_integral_630v14b5b = bool(
        np.all(
            returned_values_630v14b5b
            ==
            np.floor(
                returned_values_630v14b5b
            )
        )
    )

    expected_integral_630v14b5b = bool(
        np.all(
            expected_values_630v14b5b
            ==
            np.floor(
                expected_values_630v14b5b
            )
        )
    )

    values_exact_630v14b5b = bool(
        np.array_equal(
            returned_values_630v14b5b.astype(np.int64),
            expected_values_630v14b5b.astype(np.int64),
        )
    )

    integer_column_checks_630v14b5b[
        col_630v14b5b
    ] = {
        "finite":
            finite_630v14b5b,

        "returned_integral":
            returned_integral_630v14b5b,

        "expected_integral":
            expected_integral_630v14b5b,

        "values_exact":
            values_exact_630v14b5b,
    }


for col_630v14b5b in float_semantic_cols_630v14b5b:

    returned_values_630v14b5b = (
        returned_630v14b5b[col_630v14b5b]
        .to_numpy(dtype=np.float64)
    )

    expected_values_630v14b5b = (
        expected_630v14b5b[col_630v14b5b]
        .to_numpy(dtype=np.float64)
    )

    finite_630v14b5b = bool(
        np.all(
            np.isfinite(
                returned_values_630v14b5b
            )
        )
        and
        np.all(
            np.isfinite(
                expected_values_630v14b5b
            )
        )
    )

    values_exact_630v14b5b = bool(
        np.array_equal(
            returned_values_630v14b5b,
            expected_values_630v14b5b,
        )
    )

    float_column_checks_630v14b5b[
        col_630v14b5b
    ] = {
        "finite":
            finite_630v14b5b,

        "values_exact":
            values_exact_630v14b5b,
    }


integer_semantics_exact_630v14b5b = bool(
    all(
        entry_630v14b5b["finite"]
        and entry_630v14b5b["returned_integral"]
        and entry_630v14b5b["expected_integral"]
        and entry_630v14b5b["values_exact"]

        for entry_630v14b5b
        in integer_column_checks_630v14b5b.values()
    )
)

float_semantics_exact_630v14b5b = bool(
    all(
        entry_630v14b5b["finite"]
        and entry_630v14b5b["values_exact"]

        for entry_630v14b5b
        in float_column_checks_630v14b5b.values()
    )
)

all_stuck_info_semantics_exact_630v14b5b = bool(
    returned_schema_exact_630v14b5b
    and expected_schema_exact_630v14b5b
    and row_count_exact_630v14b5b
    and unit_id_order_exact_630v14b5b
    and integer_semantics_exact_630v14b5b
    and float_semantics_exact_630v14b5b
)

if not all_stuck_info_semantics_exact_630v14b5b:
    raise RuntimeError(
        "Resident stuck_info failed exact semantic comparison."
    )


# =============================================================================
# 5) CANONICAL SEMANTIC TABLE + HASH
# =============================================================================

canonical_stuck_info_630v14b5b = pd.DataFrame(
    {
        "unit_id":
            returned_630v14b5b[
                "unit_id"
            ].to_numpy(dtype=np.int64),

        "trajectory_length":
            returned_630v14b5b[
                "trajectory_length"
            ].to_numpy(dtype=np.int64),

        "requested_fraction":
            returned_630v14b5b[
                "requested_fraction"
            ].to_numpy(dtype=np.float64),

        "affected_count":
            returned_630v14b5b[
                "affected_count"
            ].to_numpy(dtype=np.int64),

        "actual_fraction":
            returned_630v14b5b[
                "actual_fraction"
            ].to_numpy(dtype=np.float64),

        "source_cycle":
            returned_630v14b5b[
                "source_cycle"
            ].to_numpy(dtype=np.int64),

        "stuck_start_cycle":
            returned_630v14b5b[
                "stuck_start_cycle"
            ].to_numpy(dtype=np.int64),

        "freeze_value":
            returned_630v14b5b[
                "freeze_value"
            ].to_numpy(dtype=np.float64),
    }
)

canonical_records_630v14b5b = (
    canonical_stuck_info_630v14b5b
    .to_dict(orient="records")
)

FD004_STUCK_INFO_SEMANTIC_SHA256_630V14B5B = (
    hashlib.sha256(
        json.dumps(
            canonical_records_630v14b5b,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 6) FINAL STUCK VALIDATION CONTRACT
# =============================================================================

FD004_STUCK_STRUCTURAL_VALIDATION_CONTRACT_630V14B5B = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-14B5B",

    "scenario_id":
        "FD004_S379",

    "scenario_number":
        379,

    "sensor":
        "sensor_1",

    "requested_fraction":
        0.2,

    "operator_executed_in_this_step":
        False,

    "operator_execution_source":
        "STEP_6.30V-14B5A_RESIDENT_OUTPUT",

    "engine_rows":
        int(
            len(
                returned_630v14b5b
            )
        ),

    "schema_exact":
        returned_schema_exact_630v14b5b,

    "row_count_exact":
        row_count_exact_630v14b5b,

    "engine_order_exact":
        unit_id_order_exact_630v14b5b,

    "integer_semantic_columns":
        integer_column_checks_630v14b5b,

    "float_semantic_columns":
        float_column_checks_630v14b5b,

    "integer_semantics_exact":
        integer_semantics_exact_630v14b5b,

    "float_semantics_exact":
        float_semantics_exact_630v14b5b,

    "all_stuck_info_semantics_exact":
        all_stuck_info_semantics_exact_630v14b5b,

    "stuck_mask_sha256":
        str(
            FD004_STUCK_MASK_SHA256_630V14B5A
        ),

    "stuck_info_semantic_sha256":
        FD004_STUCK_INFO_SEMANTIC_SHA256_630V14B5B,

    "stuck_core_validation_sha256":
        str(
            FD004_STUCK_CORE_VALIDATION_SHA256_630V14B5A
        ),

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "corruption_executed_in_this_step":
        False,

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_STUCK_STRUCTURAL_VALIDATION_SHA256_630V14B5B = (
    hashlib.sha256(
        json.dumps(
            FD004_STUCK_STRUCTURAL_VALIDATION_CONTRACT_630V14B5B,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) FINAL LOCK FLAGS
# =============================================================================

FD004_STUCK_STRUCTURAL_VALIDATION_LOCKED_630V14B5B = True

FD004_OPERATOR_EXECUTED_630V14B5B = False
FD004_CORRUPTION_EXECUTED_630V14B5B = False

FD004_FULL_CORRUPTION_EXECUTED_630V14B5B = False

FD004_PREPROCESSING_RERUN_630V14B5B = False
FD004_MODEL_INFERENCE_PERFORMED_630V14B5B = False
FD004_MC_EXECUTION_PERFORMED_630V14B5B = False

FD004_DEV_TARGETS_ERRORS_USED_630V14B5B = False
FD004_THRESHOLD_SELECTED_630V14B5B = False
FD004_POLICY_SELECTED_630V14B5B = False

FD004_TEST_CONTENTS_READ_630V14B5B = False
FD004_RUL_CONTENTS_READ_630V14B5B = False

all_checks_passed_630v14b5b = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-14B5B — FD004 STUCK INFO SEMANTIC "
    "VALIDATION + FINAL STUCK LOCK"
)

print("=" * 176)

print("Operator executed this step       : False")
print("Corruption executed this step     : False")
print("Resident B5A result reused        : True")
print("Full 381-scenario run executed    : False")
print("Preprocessing rerun               : False")
print("Model inference performed         : False")
print("MC execution performed            : False")
print("DEV targets/errors used           : False")
print("Threshold selected                : False")
print("Policy selected                   : False")
print("TEST contents read                : False")
print("Official RUL contents read        : False")

print("\n--- STUCK INFO AUTHORITY ---")

print(
    "Returned schema exact             :",
    returned_schema_exact_630v14b5b,
)

print(
    "Expected schema exact             :",
    expected_schema_exact_630v14b5b,
)

print(
    "50 engine rows exact              :",
    row_count_exact_630v14b5b,
)

print(
    "Engine row order exact            :",
    unit_id_order_exact_630v14b5b,
)

print("\n--- INTEGER-SEMANTIC COLUMNS ---")

for (
    col_630v14b5b,
    checks_630v14b5b,
) in integer_column_checks_630v14b5b.items():

    print(
        f"{col_630v14b5b:24s}:",
        checks_630v14b5b["values_exact"],
    )

print("\n--- FLOAT-SEMANTIC COLUMNS ---")

for (
    col_630v14b5b,
    checks_630v14b5b,
) in float_column_checks_630v14b5b.items():

    print(
        f"{col_630v14b5b:24s}:",
        checks_630v14b5b["values_exact"],
    )

print("\n--- FINAL SEMANTIC RESULT ---")

print(
    "Integer semantics exact           :",
    integer_semantics_exact_630v14b5b,
)

print(
    "Float semantics exact             :",
    float_semantics_exact_630v14b5b,
)

print(
    "All stuck_info semantics exact    :",
    all_stuck_info_semantics_exact_630v14b5b,
)

print("\n--- EVIDENCE FINGERPRINTS ---")

print(
    "Stuck mask SHA-256                :",
    FD004_STUCK_MASK_SHA256_630V14B5A,
)

print(
    "Stuck info semantic SHA-256       :",
    FD004_STUCK_INFO_SEMANTIC_SHA256_630V14B5B,
)

print(
    "Final stuck structural SHA-256    :",
    FD004_STUCK_STRUCTURAL_VALIDATION_SHA256_630V14B5B,
)

print(
    "\nPASS — all 50 returned stuck_info records exactly match "
    "the independently reconstructed historical terminal-freeze "
    "semantics. No operator was rerun in this step."
)

print(
    "Next: close the five-family structural validation evidence, "
    "then audit the historical manifest dispatcher before any "
    "381-scenario execution."
)

In [ ]:
# STEP 6.30V-15 — FD004 FIVE-FAMILY STRUCTURAL VALIDATION EVIDENCE CLOSURE
#
# Evidence closure only.
# NO corruption/operator execution.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import hashlib
import json

required_630v15 = (
    "all_checks_passed_630v14b1",
    "all_checks_passed_630v14b2",
    "all_checks_passed_630v14b3",
    "all_checks_passed_630v14b4",
    "all_checks_passed_630v14b5b",
    "FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A",
    "FD004_OPERATOR_CALL_SEMANTICS_AUDIT_SHA256_630V14B0",
    "FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1",
    "FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2",
    "FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3",
    "FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4",
    "FD004_STUCK_STRUCTURAL_VALIDATION_SHA256_630V14B5B",
)

missing_630v15 = [
    name_630v15
    for name_630v15 in required_630v15
    if name_630v15 not in globals()
]

if missing_630v15:
    raise RuntimeError(
        "Missing STEP 6.30V-15 authority object(s): "
        + ", ".join(missing_630v15)
    )

prior_pass_flags_630v15 = {
    "bias": bool(all_checks_passed_630v14b1),
    "noise": bool(all_checks_passed_630v14b2),
    "drift": bool(all_checks_passed_630v14b3),
    "dropout": bool(all_checks_passed_630v14b4),
    "stuck": bool(all_checks_passed_630v14b5b),
}

if not all(prior_pass_flags_630v15.values()):
    raise RuntimeError(
        "One or more five-family structural validations are not passed."
    )

expected_hashes_630v15 = {
    "row_identity_audit":
        "749a4f8a3b51e5309d1c79375846f75129425be93ca913ced2ef3441053c75fa",

    "positional_operator_contract":
        "6515543a32441fb32454d62930df802dfd89763ae0c7bf2d50e0e9c7c11c9d23",

    "structural_preflight":
        "b3f15e5f59deedb9ae02b5cc23e887570891a038ae30f99643e35085ecbddd61",

    "call_semantics_audit":
        "8d04939f9b9737d5b96adda2489084125a6ca7abe7e6138dd6a40673e2ccd8c6",

    "bias":
        "c4aba4f9cce4bb3bd0766dcf7601c32380c96e3ca760b523f94f573575b3a804",

    "noise":
        "4dc947759dff80224f686a56661f7b3ef8e1899554c25fa41a7cca2af0a4f23e",

    "drift":
        "0b3bcebb26d7090d25a1022867eb792835af1d89ed2952892ace35a8d3e3d0be",

    "dropout":
        "9abd3e5544d5997047974609be3bddb38eb6e00e502e6bf1d9a2121a1abfe873",

    "stuck":
        "674ce373ebb9654602175ab38e0d484a8bed749f06dfe3cfc35a2df5e7f03ce2",
}

observed_hashes_630v15 = {
    "row_identity_audit":
        str(FD004_ROW_IDENTITY_AUDIT_SHA256_630V12R2),

    "positional_operator_contract":
        str(FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13),

    "structural_preflight":
        str(FD004_STRUCTURAL_VALIDATION_PREFLIGHT_SHA256_630V14A),

    "call_semantics_audit":
        str(FD004_OPERATOR_CALL_SEMANTICS_AUDIT_SHA256_630V14B0),

    "bias":
        str(FD004_BIAS_STRUCTURAL_VALIDATION_SHA256_630V14B1),

    "noise":
        str(FD004_NOISE_STRUCTURAL_VALIDATION_SHA256_630V14B2),

    "drift":
        str(FD004_DRIFT_STRUCTURAL_VALIDATION_SHA256_630V14B3),

    "dropout":
        str(FD004_DROPOUT_STRUCTURAL_VALIDATION_SHA256_630V14B4),

    "stuck":
        str(FD004_STUCK_STRUCTURAL_VALIDATION_SHA256_630V14B5B),
}

hash_checks_630v15 = {
    key_630v15: bool(
        observed_hashes_630v15[key_630v15]
        == expected_hashes_630v15[key_630v15]
    )
    for key_630v15
    in expected_hashes_630v15
}

if not all(hash_checks_630v15.values()):
    failed_630v15 = [
        key_630v15
        for key_630v15, passed_630v15
        in hash_checks_630v15.items()
        if not passed_630v15
    ]

    raise RuntimeError(
        "Frozen structural evidence fingerprint mismatch: "
        + ", ".join(failed_630v15)
    )

FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_630V15 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-15",

    "status":
        "FIVE_FAMILY_STRUCTURAL_VALIDATION_CLOSED",

    "validated_families":
        (
            "bias",
            "noise",
            "drift",
            "dropout",
            "stuck",
        ),

    "validation_scenarios":
        {
            "bias": "FD004_S007",
            "noise": "FD004_S130",
            "drift": "FD004_S196",
            "dropout": "FD004_S316",
            "stuck": "FD004_S379",
        },

    "row_identity_authority":
        "UNIT_CYCLE_ROWWISE_ORDER",

    "scientific_corruption_semantics_changed":
        False,

    "implementation_bridge_correction":
        (
            "local reset_index(drop=True) positional projection "
            "followed by restoration of caller index"
        ),

    "evidence_sha256":
        dict(observed_hashes_630v15),

    "all_evidence_hashes_exact":
        True,

    "all_five_families_structurally_validated":
        True,

    "operator_executed_in_this_step":
        False,

    "corruption_executed_in_this_step":
        False,

    "full_381_scenario_execution_started":
        False,

    "preprocessing_rerun":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "next_required_gate":
        (
            "exact historical manifest-row dispatcher recovery/audit "
            "before any 381-scenario execution"
        ),
}

FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15 = (
    hashlib.sha256(
        json.dumps(
            FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CONTRACT_630V15,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()
)

FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_CLOSED_630V15 = True
FD004_FULL_CORRUPTION_EXECUTION_ADMITTED_630V15 = False

FD004_OPERATOR_EXECUTED_630V15 = False
FD004_CORRUPTION_EXECUTED_630V15 = False
FD004_MODEL_INFERENCE_PERFORMED_630V15 = False
FD004_MC_EXECUTION_PERFORMED_630V15 = False
FD004_DEV_TARGETS_ERRORS_USED_630V15 = False
FD004_THRESHOLD_SELECTED_630V15 = False
FD004_POLICY_SELECTED_630V15 = False
FD004_TEST_CONTENTS_READ_630V15 = False
FD004_RUL_CONTENTS_READ_630V15 = False

all_checks_passed_630v15 = True

print("=" * 176)

print(
    "STEP 6.30V-15 — FD004 FIVE-FAMILY STRUCTURAL "
    "VALIDATION EVIDENCE CLOSURE"
)

print("=" * 176)

print("Operator executed this step       :", False)
print("Corruption executed this step     :", False)
print("Full 381-scenario run started     :", False)
print("Preprocessing rerun               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FIVE-FAMILY VALIDATION STATUS ---")

for family_630v15 in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    print(
        f"{family_630v15:8s}:",
        prior_pass_flags_630v15[
            family_630v15
        ],
    )

print("\n--- FROZEN EVIDENCE FINGERPRINT CHECKS ---")

for key_630v15 in (
    "row_identity_audit",
    "positional_operator_contract",
    "structural_preflight",
    "call_semantics_audit",
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    print(
        f"{key_630v15:30s}:",
        hash_checks_630v15[
            key_630v15
        ],
    )

print("\n--- CLOSURE ---")

print(
    "All five families validated       :",
    True,
)

print(
    "Scientific semantics changed      :",
    False,
)

print(
    "381-scenario execution admitted   :",
    False,
)

print(
    "Five-family closure SHA-256       :",
    FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15,
)

print(
    "\nPASS — bias, noise, drift, dropout, and stuck structural "
    "semantics are now evidence-closed for FD004."
)

print(
    "Next required gate: recover and audit the exact historical "
    "manifest-row dispatcher before any 381-scenario execution."
)

In [ ]:
# STEP 6.30V-16A — FD004 HISTORICAL MANIFEST-DISPATCHER SOURCE LOCATOR
#
# SOURCE / METADATA AUDIT ONLY.
# NO dispatcher execution.
# NO corruption.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import hashlib
import inspect


# =============================================================================
# 0) PRECONDITION
# =============================================================================

if "all_checks_passed_630v15" not in globals():
    raise RuntimeError(
        "STEP 6.30V-15 authority is missing."
    )

if not bool(all_checks_passed_630v15):
    raise RuntimeError(
        "STEP 6.30V-15 must remain passed."
    )


# =============================================================================
# 1) FROZEN HISTORICAL TARGET
# =============================================================================

TARGET_FUNCTION_630V16A = (
    "apply_manifest_scenario_628ha"
)

EXPECTED_DISPATCHER_CELL_SHA256_630V16A = (
    "51055c9790966038aea53e473ca46cb97"
    "db18784f351cb3cc105894e9e514dfe"
)


# =============================================================================
# 2) SAFE SEARCH HELPERS
# =============================================================================

def _sha_text_630v16a(text_630v16a):
    return hashlib.sha256(
        text_630v16a.encode("utf-8")
    ).hexdigest()


def _safe_search_nested_630v16a(
    obj_630v16a,
    path_630v16a,
    findings_630v16a,
    depth_630v16a=0,
    max_depth_630v16a=5,
    node_counter_630v16a=None,
    max_nodes_630v16a=20000,
):
    if node_counter_630v16a is None:
        node_counter_630v16a = [0]

    if node_counter_630v16a[0] >= max_nodes_630v16a:
        return

    node_counter_630v16a[0] += 1

    if depth_630v16a > max_depth_630v16a:
        return

    if isinstance(obj_630v16a, str):
        if TARGET_FUNCTION_630V16A in obj_630v16a:
            findings_630v16a.append(
                {
                    "path":
                        path_630v16a,

                    "text":
                        obj_630v16a,

                    "length":
                        len(obj_630v16a),

                    "sha256":
                        _sha_text_630v16a(
                            obj_630v16a
                        ),
                }
            )
        return

    if isinstance(
        obj_630v16a,
        (list, tuple),
    ):
        for idx_630v16a, item_630v16a in enumerate(
            obj_630v16a
        ):
            _safe_search_nested_630v16a(
                item_630v16a,
                f"{path_630v16a}[{idx_630v16a}]",
                findings_630v16a,
                depth_630v16a + 1,
                max_depth_630v16a,
                node_counter_630v16a,
                max_nodes_630v16a,
            )
        return

    if isinstance(obj_630v16a, dict):
        for (
            key_630v16a,
            value_630v16a,
        ) in obj_630v16a.items():

            key_text_630v16a = str(
                key_630v16a
            )

            _safe_search_nested_630v16a(
                value_630v16a,
                (
                    f"{path_630v16a}"
                    f"[{key_text_630v16a!r}]"
                ),
                findings_630v16a,
                depth_630v16a + 1,
                max_depth_630v16a,
                node_counter_630v16a,
                max_nodes_630v16a,
            )


# =============================================================================
# 3) FIRST: CHECK WHETHER HISTORICAL CALLABLE ITSELF IS RESIDENT
#    inspect only; never call it.
# =============================================================================

resident_callable_present_630v16a = bool(
    TARGET_FUNCTION_630V16A
    in globals()
    and callable(
        globals()[
            TARGET_FUNCTION_630V16A
        ]
    )
)

resident_callable_source_630v16a = None
resident_callable_source_sha256_630v16a = None
resident_callable_signature_630v16a = None

if resident_callable_present_630v16a:
    resident_callable_630v16a = globals()[
        TARGET_FUNCTION_630V16A
    ]

    resident_callable_signature_630v16a = str(
        inspect.signature(
            resident_callable_630v16a
        )
    )

    try:
        resident_callable_source_630v16a = (
            inspect.getsource(
                resident_callable_630v16a
            )
        )

        resident_callable_source_sha256_630v16a = (
            _sha_text_630v16a(
                resident_callable_source_630v16a
            )
        )

    except Exception:
        resident_callable_source_630v16a = None
        resident_callable_source_sha256_630v16a = None


# =============================================================================
# 4) IDENTIFY LIKELY SOURCE-CONTAINER GLOBALS WITHOUT ASSUMING A NAME
# =============================================================================

name_tokens_630v16a = (
    "cell",
    "source",
    "notebook",
    "630tr",
    "628h",
    "dispatch",
    "manifest",
    "history",
)

candidate_root_names_630v16a = sorted(
    name_630v16a
    for name_630v16a in globals().keys()
    if any(
        token_630v16a in name_630v16a.lower()
        for token_630v16a
        in name_tokens_630v16a
    )
)

# Do not recursively traverse modules, functions, DataFrames,
# tensors, models, etc. Only simple source-like containers.
searchable_roots_630v16a = []

for name_630v16a in candidate_root_names_630v16a:
    obj_630v16a = globals()[
        name_630v16a
    ]

    if isinstance(
        obj_630v16a,
        (str, list, tuple, dict),
    ):
        searchable_roots_630v16a.append(
            name_630v16a
        )


# =============================================================================
# 5) SAFE NESTED SEARCH
# =============================================================================

dispatcher_source_findings_630v16a = []

for root_name_630v16a in searchable_roots_630v16a:

    root_obj_630v16a = globals()[
        root_name_630v16a
    ]

    _safe_search_nested_630v16a(
        root_obj_630v16a,
        root_name_630v16a,
        dispatcher_source_findings_630v16a,
    )


# De-duplicate identical path/text pairs only.
deduped_findings_630v16a = []
_seen_630v16a = set()

for finding_630v16a in dispatcher_source_findings_630v16a:

    dedupe_key_630v16a = (
        finding_630v16a["path"],
        finding_630v16a["sha256"],
    )

    if dedupe_key_630v16a not in _seen_630v16a:
        _seen_630v16a.add(
            dedupe_key_630v16a
        )

        deduped_findings_630v16a.append(
            finding_630v16a
        )

dispatcher_source_findings_630v16a = (
    deduped_findings_630v16a
)


# =============================================================================
# 6) EXACT HISTORICAL CELL-FINGERPRINT MATCH
# =============================================================================

exact_cell_matches_630v16a = [
    finding_630v16a
    for finding_630v16a
    in dispatcher_source_findings_630v16a
    if finding_630v16a["sha256"]
    ==
    EXPECTED_DISPATCHER_CELL_SHA256_630V16A
]

exact_historical_cell_recovered_630v16a = bool(
    len(
        exact_cell_matches_630v16a
    ) >= 1
)

if len(exact_cell_matches_630v16a) > 1:

    exact_texts_630v16a = {
        item_630v16a["text"]
        for item_630v16a
        in exact_cell_matches_630v16a
    }

    if len(exact_texts_630v16a) != 1:
        raise RuntimeError(
            "Multiple non-identical source texts match the same "
            "historical dispatcher-cell SHA-256."
        )


# =============================================================================
# 7) RETAIN LOCATOR EVIDENCE — NO EXECUTION
# =============================================================================

FD004_HISTORICAL_DISPATCHER_SOURCE_FINDINGS_630V16A = tuple(
    {
        "path":
            finding_630v16a["path"],

        "length":
            int(
                finding_630v16a["length"]
            ),

        "sha256":
            finding_630v16a["sha256"],

        "exact_historical_cell_sha_match":
            bool(
                finding_630v16a["sha256"]
                ==
                EXPECTED_DISPATCHER_CELL_SHA256_630V16A
            ),
    }
    for finding_630v16a
    in dispatcher_source_findings_630v16a
)

if exact_historical_cell_recovered_630v16a:
    FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A = (
        exact_cell_matches_630v16a[0][
            "text"
        ]
    )
else:
    FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A = None


FD004_HISTORICAL_DISPATCHER_LOCATOR_630V16A = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16A",

    "target_function":
        TARGET_FUNCTION_630V16A,

    "expected_historical_dispatcher_cell_sha256":
        EXPECTED_DISPATCHER_CELL_SHA256_630V16A,

    "resident_callable_present":
        resident_callable_present_630v16a,

    "resident_callable_signature":
        resident_callable_signature_630v16a,

    "resident_callable_source_sha256":
        resident_callable_source_sha256_630v16a,

    "candidate_root_names":
        tuple(
            candidate_root_names_630v16a
        ),

    "searchable_source_roots":
        tuple(
            searchable_roots_630v16a
        ),

    "number_of_source_findings":
        int(
            len(
                dispatcher_source_findings_630v16a
            )
        ),

    "number_of_exact_cell_sha_matches":
        int(
            len(
                exact_cell_matches_630v16a
            )
        ),

    "exact_historical_cell_recovered":
        exact_historical_cell_recovered_630v16a,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_HISTORICAL_DISPATCHER_LOCATOR_SHA256_630V16A = (
    hashlib.sha256(
        repr(
            FD004_HISTORICAL_DISPATCHER_LOCATOR_630V16A
        ).encode("utf-8")
    ).hexdigest()
)

FD004_DISPATCHER_EXECUTED_630V16A = False
FD004_CORRUPTION_EXECUTED_630V16A = False
FD004_MODEL_INFERENCE_PERFORMED_630V16A = False
FD004_MC_EXECUTION_PERFORMED_630V16A = False
FD004_DEV_TARGETS_ERRORS_USED_630V16A = False
FD004_THRESHOLD_SELECTED_630V16A = False
FD004_POLICY_SELECTED_630V16A = False
FD004_TEST_CONTENTS_READ_630V16A = False
FD004_RUL_CONTENTS_READ_630V16A = False

all_checks_passed_630v16a = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-16A — FD004 HISTORICAL "
    "MANIFEST-DISPATCHER SOURCE LOCATOR"
)

print("=" * 176)

print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- HISTORICAL TARGET ---")

print(
    "Target function                   :",
    TARGET_FUNCTION_630V16A,
)

print(
    "Expected dispatcher-cell SHA-256  :",
    EXPECTED_DISPATCHER_CELL_SHA256_630V16A,
)

print("\n--- RESIDENT CALLABLE CHECK ---")

print(
    "Historical callable resident      :",
    resident_callable_present_630v16a,
)

print(
    "Resident signature                :",
    resident_callable_signature_630v16a,
)

print(
    "Resident source SHA-256           :",
    resident_callable_source_sha256_630v16a,
)

print("\n--- SOURCE-CONTAINER SEARCH ---")

print(
    "Candidate named globals           :",
    len(
        candidate_root_names_630v16a
    ),
)

print(
    "Searchable source containers      :",
    tuple(
        searchable_roots_630v16a
    ),
)

print(
    "Dispatcher-containing texts found :",
    len(
        dispatcher_source_findings_630v16a
    ),
)

for idx_630v16a, finding_630v16a in enumerate(
    dispatcher_source_findings_630v16a,
    start=1,
):
    print(
        f"\nFinding {idx_630v16a}"
    )

    print(
        "  path                            :",
        finding_630v16a["path"],
    )

    print(
        "  text length                     :",
        finding_630v16a["length"],
    )

    print(
        "  SHA-256                         :",
        finding_630v16a["sha256"],
    )

    print(
        "  exact historical cell match     :",
        finding_630v16a["sha256"]
        ==
        EXPECTED_DISPATCHER_CELL_SHA256_630V16A,
    )

print("\n--- LOCATOR RESULT ---")

print(
    "Exact dispatcher cell matches     :",
    len(
        exact_cell_matches_630v16a
    ),
)

print(
    "Exact historical cell recovered   :",
    exact_historical_cell_recovered_630v16a,
)

print(
    "Locator evidence SHA-256          :",
    FD004_HISTORICAL_DISPATCHER_LOCATOR_SHA256_630V16A,
)

print(
    "\nPASS — this step only located resident historical "
    "dispatcher-source evidence. Nothing was executed."
)

if exact_historical_cell_recovered_630v16a:
    print(
        "Next: extract and audit the exact dispatcher definition "
        "from the recovered fingerprint-matched historical cell."
    )
else:
    print(
        "Next: use the locator findings to recover the dispatcher "
        "definition without guessing or executing anything."
    )

In [ ]:
# STEP 6.30V-16B — FD004 EXACT HISTORICAL DISPATCHER
# DEFINITION EXTRACTION + STATIC AUDIT
#
# SOURCE-ONLY.
# NO exec().
# NO dispatcher call.
# NO corruption.
# NO preprocessing.
# NO model inference / MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import ast
import hashlib
import json
import re


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v16b = (
    "all_checks_passed_630v16a",
    "FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A",
    "FD004_HISTORICAL_DISPATCHER_LOCATOR_SHA256_630V16A",
)

missing_630v16b = [
    name_630v16b
    for name_630v16b in required_630v16b
    if name_630v16b not in globals()
]

if missing_630v16b:
    raise RuntimeError(
        "Missing STEP 6.30V-16B authority object(s): "
        + ", ".join(missing_630v16b)
    )

if not bool(all_checks_passed_630v16a):
    raise RuntimeError(
        "STEP 6.30V-16A must remain passed."
    )

historical_cell_source_630v16b = (
    FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A
)

if not isinstance(
    historical_cell_source_630v16b,
    str,
):
    raise RuntimeError(
        "Recovered historical dispatcher cell source is not text."
    )

TARGET_FUNCTION_630V16B = (
    "apply_manifest_scenario_628ha"
)

EXPECTED_CELL_SHA256_630V16B = (
    "51055c9790966038aea53e473ca46cb97"
    "db18784f351cb3cc105894e9e514dfe"
)

observed_cell_sha256_630v16b = (
    hashlib.sha256(
        historical_cell_source_630v16b.encode("utf-8")
    ).hexdigest()
)

if (
    observed_cell_sha256_630v16b
    != EXPECTED_CELL_SHA256_630V16B
):
    raise RuntimeError(
        "Recovered historical dispatcher cell fingerprint changed."
    )


# =============================================================================
# 1) PARSE HISTORICAL CELL — STATIC ONLY
# =============================================================================

try:
    tree_630v16b = ast.parse(
        historical_cell_source_630v16b
    )
except SyntaxError as exc_630v16b:
    raise RuntimeError(
        "Fingerprint-matched historical dispatcher cell could not "
        "be parsed as Python source."
    ) from exc_630v16b


target_nodes_630v16b = [
    node_630v16b
    for node_630v16b in ast.walk(
        tree_630v16b
    )
    if isinstance(
        node_630v16b,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    )
    and node_630v16b.name
    == TARGET_FUNCTION_630V16B
]

if len(target_nodes_630v16b) != 1:
    raise RuntimeError(
        "Expected exactly one historical dispatcher definition; "
        f"found {len(target_nodes_630v16b)}."
    )

dispatcher_node_630v16b = (
    target_nodes_630v16b[0]
)

if not hasattr(
    dispatcher_node_630v16b,
    "end_lineno",
):
    raise RuntimeError(
        "Python AST does not expose end_lineno for exact extraction."
    )


# =============================================================================
# 2) EXTRACT EXACT FUNCTION SOURCE
# =============================================================================

cell_lines_630v16b = (
    historical_cell_source_630v16b.splitlines(
        keepends=True
    )
)

start_line_630v16b = int(
    dispatcher_node_630v16b.lineno
)

end_line_630v16b = int(
    dispatcher_node_630v16b.end_lineno
)

dispatcher_source_630v16b = "".join(
    cell_lines_630v16b[
        start_line_630v16b - 1:
        end_line_630v16b
    ]
)

if TARGET_FUNCTION_630V16B not in dispatcher_source_630v16b:
    raise RuntimeError(
        "Exact extracted source does not contain target function name."
    )

dispatcher_source_sha256_630v16b = (
    hashlib.sha256(
        dispatcher_source_630v16b.encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 3) STATIC SIGNATURE EXTRACTION — NO FUNCTION CREATION
# =============================================================================

args_node_630v16b = (
    dispatcher_node_630v16b.args
)

positional_args_630v16b = (
    list(
        args_node_630v16b.posonlyargs
    )
    +
    list(
        args_node_630v16b.args
    )
)

positional_arg_names_630v16b = tuple(
    arg_630v16b.arg
    for arg_630v16b
    in positional_args_630v16b
)

kwonly_arg_names_630v16b = tuple(
    arg_630v16b.arg
    for arg_630v16b
    in args_node_630v16b.kwonlyargs
)

vararg_name_630v16b = (
    None
    if args_node_630v16b.vararg is None
    else args_node_630v16b.vararg.arg
)

kwarg_name_630v16b = (
    None
    if args_node_630v16b.kwarg is None
    else args_node_630v16b.kwarg.arg
)

defaults_source_630v16b = tuple(
    ast.get_source_segment(
        historical_cell_source_630v16b,
        default_630v16b,
    )
    for default_630v16b
    in args_node_630v16b.defaults
)

kw_defaults_source_630v16b = tuple(
    None
    if default_630v16b is None
    else ast.get_source_segment(
        historical_cell_source_630v16b,
        default_630v16b,
    )
    for default_630v16b
    in args_node_630v16b.kw_defaults
)


# =============================================================================
# 4) STATIC CALL AUDIT
# =============================================================================

def _call_name_630v16b(
    call_node_630v16b,
):
    fn_630v16b = call_node_630v16b.func

    if isinstance(
        fn_630v16b,
        ast.Name,
    ):
        return fn_630v16b.id

    if isinstance(
        fn_630v16b,
        ast.Attribute,
    ):
        parts_630v16b = []
        cur_630v16b = fn_630v16b

        while isinstance(
            cur_630v16b,
            ast.Attribute,
        ):
            parts_630v16b.append(
                cur_630v16b.attr
            )
            cur_630v16b = cur_630v16b.value

        if isinstance(
            cur_630v16b,
            ast.Name,
        ):
            parts_630v16b.append(
                cur_630v16b.id
            )

        return ".".join(
            reversed(
                parts_630v16b
            )
        )

    return ast.dump(
        fn_630v16b,
        include_attributes=False,
    )


call_records_630v16b = []

for node_630v16b in ast.walk(
    dispatcher_node_630v16b
):
    if isinstance(
        node_630v16b,
        ast.Call,
    ):
        source_segment_630v16b = (
            ast.get_source_segment(
                historical_cell_source_630v16b,
                node_630v16b,
            )
        )

        call_records_630v16b.append(
            {
                "line":
                    int(
                        node_630v16b.lineno
                    ),

                "call_name":
                    _call_name_630v16b(
                        node_630v16b
                    ),

                "source":
                    source_segment_630v16b,
            }
        )

call_records_630v16b = tuple(
    sorted(
        call_records_630v16b,
        key=lambda x_630v16b: (
            x_630v16b["line"],
            x_630v16b["call_name"],
        ),
    )
)


# =============================================================================
# 5) STATIC IF / ELIF TEST AUDIT
# =============================================================================

branch_records_630v16b = []

for node_630v16b in ast.walk(
    dispatcher_node_630v16b
):
    if isinstance(
        node_630v16b,
        ast.If,
    ):
        test_source_630v16b = (
            ast.get_source_segment(
                historical_cell_source_630v16b,
                node_630v16b.test,
            )
        )

        branch_records_630v16b.append(
            {
                "line":
                    int(
                        node_630v16b.lineno
                    ),

                "test":
                    test_source_630v16b,
            }
        )

branch_records_630v16b = tuple(
    sorted(
        branch_records_630v16b,
        key=lambda x_630v16b: x_630v16b[
            "line"
        ],
    )
)


# =============================================================================
# 6) REFERENCED IDENTIFIERS
# =============================================================================

loaded_names_630v16b = tuple(
    sorted(
        {
            node_630v16b.id
            for node_630v16b
            in ast.walk(
                dispatcher_node_630v16b
            )
            if isinstance(
                node_630v16b,
                ast.Name,
            )
            and isinstance(
                node_630v16b.ctx,
                ast.Load,
            )
        }
    )
)


# =============================================================================
# 7) DISCOVER RESIDENT EXPECTED DISPATCHER-SOURCE HASH AUTHORITIES
#    Do not assume a variable name or expected value.
# =============================================================================

hex64_pattern_630v16b = re.compile(
    r"^[0-9a-fA-F]{64}$"
)

resident_dispatcher_hash_authorities_630v16b = {}

for name_630v16b, value_630v16b in list(
    globals().items()
):
    upper_name_630v16b = (
        str(name_630v16b).upper()
    )

    if (
        "DISPATCHER" in upper_name_630v16b
        and
        "SOURCE" in upper_name_630v16b
        and
        "SHA256" in upper_name_630v16b
        and
        isinstance(
            value_630v16b,
            str,
        )
        and
        hex64_pattern_630v16b.fullmatch(
            value_630v16b
        )
    ):
        resident_dispatcher_hash_authorities_630v16b[
            str(name_630v16b)
        ] = str(
            value_630v16b
        )

resident_dispatcher_hash_matches_630v16b = {
    name_630v16b:
        bool(
            value_630v16b
            ==
            dispatcher_source_sha256_630v16b
        )
    for (
        name_630v16b,
        value_630v16b,
    )
    in resident_dispatcher_hash_authorities_630v16b.items()
}


# =============================================================================
# 8) LOCK STATIC EXTRACTION EVIDENCE
# =============================================================================

FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_630V16B = (
    dispatcher_source_630v16b
)

FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B = (
    dispatcher_source_sha256_630v16b
)

FD004_HISTORICAL_DISPATCHER_AST_AUDIT_630V16B = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16B",

    "target_function":
        TARGET_FUNCTION_630V16B,

    "historical_cell_sha256":
        observed_cell_sha256_630v16b,

    "function_start_line":
        start_line_630v16b,

    "function_end_line":
        end_line_630v16b,

    "function_source_sha256":
        dispatcher_source_sha256_630v16b,

    "positional_args":
        positional_arg_names_630v16b,

    "positional_defaults":
        defaults_source_630v16b,

    "keyword_only_args":
        kwonly_arg_names_630v16b,

    "keyword_only_defaults":
        kw_defaults_source_630v16b,

    "vararg":
        vararg_name_630v16b,

    "kwarg":
        kwarg_name_630v16b,

    "branches":
        branch_records_630v16b,

    "calls":
        call_records_630v16b,

    "loaded_names":
        loaded_names_630v16b,

    "resident_dispatcher_source_hash_authorities":
        resident_dispatcher_hash_authorities_630v16b,

    "resident_dispatcher_source_hash_matches":
        resident_dispatcher_hash_matches_630v16b,

    "source_executed":
        False,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_HISTORICAL_DISPATCHER_AST_AUDIT_SHA256_630V16B = (
    hashlib.sha256(
        json.dumps(
            FD004_HISTORICAL_DISPATCHER_AST_AUDIT_630V16B,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DISPATCHER_SOURCE_EXECUTED_630V16B = False
FD004_DISPATCHER_EXECUTED_630V16B = False
FD004_CORRUPTION_EXECUTED_630V16B = False
FD004_MODEL_INFERENCE_PERFORMED_630V16B = False
FD004_MC_EXECUTION_PERFORMED_630V16B = False
FD004_DEV_TARGETS_ERRORS_USED_630V16B = False
FD004_THRESHOLD_SELECTED_630V16B = False
FD004_POLICY_SELECTED_630V16B = False
FD004_TEST_CONTENTS_READ_630V16B = False
FD004_RUL_CONTENTS_READ_630V16B = False

all_checks_passed_630v16b = True


# =============================================================================
# 9) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-16B — FD004 EXACT HISTORICAL DISPATCHER "
    "DEFINITION EXTRACTION + STATIC AUDIT"
)

print("=" * 176)

print("Historical source executed        :", False)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- EXACT DEFINITION AUTHORITY ---")

print(
    "Target function                   :",
    TARGET_FUNCTION_630V16B,
)

print(
    "Cell SHA-256 exact                :",
    observed_cell_sha256_630v16b
    == EXPECTED_CELL_SHA256_630V16B,
)

print(
    "Historical cell lines             :",
    (
        start_line_630v16b,
        end_line_630v16b,
    ),
)

print(
    "Extracted function chars          :",
    len(
        dispatcher_source_630v16b
    ),
)

print(
    "Extracted function SHA-256        :",
    dispatcher_source_sha256_630v16b,
)

print("\n--- STATIC SIGNATURE ---")

print(
    "Positional arguments              :",
    positional_arg_names_630v16b,
)

print(
    "Positional defaults               :",
    defaults_source_630v16b,
)

print(
    "Keyword-only arguments            :",
    kwonly_arg_names_630v16b,
)

print(
    "Keyword-only defaults             :",
    kw_defaults_source_630v16b,
)

print(
    "*args                             :",
    vararg_name_630v16b,
)

print(
    "**kwargs                          :",
    kwarg_name_630v16b,
)

print("\n--- IF / ELIF TESTS ---")

for record_630v16b in branch_records_630v16b:
    print(
        f"line {record_630v16b['line']:4d}: "
        f"{record_630v16b['test']}"
    )

print("\n--- CALL EXPRESSIONS ---")

for record_630v16b in call_records_630v16b:
    print(
        f"line {record_630v16b['line']:4d}: "
        f"{record_630v16b['source']}"
    )

print("\n--- RESIDENT EXPECTED SOURCE-HASH AUTHORITIES ---")

if resident_dispatcher_hash_authorities_630v16b:
    for (
        name_630v16b,
        value_630v16b,
    ) in sorted(
        resident_dispatcher_hash_authorities_630v16b.items()
    ):
        print(
            f"{name_630v16b}"
        )
        print(
            "  value                           :",
            value_630v16b,
        )
        print(
            "  matches extracted function      :",
            resident_dispatcher_hash_matches_630v16b[
                name_630v16b
            ],
        )
else:
    print(
        "No resident dispatcher-function source-hash "
        "authority discovered."
    )

print("\n--- EXACT HISTORICAL FUNCTION SOURCE ---")

for (
    relative_line_630v16b,
    text_line_630v16b,
) in enumerate(
    dispatcher_source_630v16b.splitlines(),
    start=1,
):
    print(
        f"{relative_line_630v16b:03d} | "
        f"{text_line_630v16b}"
    )

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Dispatcher AST audit SHA-256      :",
    FD004_HISTORICAL_DISPATCHER_AST_AUDIT_SHA256_630V16B,
)

print(
    "\nPASS — one exact historical dispatcher definition was "
    "extracted from the fingerprint-matched source cell and audited "
    "statically. It was not created, executed, or adapted."
)

print(
    "Next: derive the FD004 dispatcher adaptation strictly from this "
    "printed historical definition and the already-validated FD004 "
    "operator registry."
)

In [ ]:
# STEP 6.30V-16C — FD004 HISTORICAL DISPATCHER CALL-SITE /
# base_raw + regime_id CONTEXT AUDIT
#
# SOURCE ONLY.
# NO exec().
# NO dispatcher execution.
# NO corruption.
# NO preprocessing.
# NO model inference / MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import ast
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v16c = (
    "all_checks_passed_630v16b",
    "FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A",
    "FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B",
    "FD004_HISTORICAL_DISPATCHER_AST_AUDIT_SHA256_630V16B",
)

missing_630v16c = [
    name_630v16c
    for name_630v16c in required_630v16c
    if name_630v16c not in globals()
]

if missing_630v16c:
    raise RuntimeError(
        "Missing STEP 6.30V-16C authority object(s): "
        + ", ".join(missing_630v16c)
    )

if not bool(all_checks_passed_630v16b):
    raise RuntimeError(
        "STEP 6.30V-16B must remain passed."
    )

historical_source_630v16c = (
    FD004_HISTORICAL_DISPATCHER_CELL_SOURCE_630V16A
)

if not isinstance(
    historical_source_630v16c,
    str,
):
    raise RuntimeError(
        "Historical dispatcher cell source authority is not text."
    )

TARGET_630V16C = (
    "apply_manifest_scenario_628ha"
)


# =============================================================================
# 1) PARSE SOURCE — STATIC ONLY
# =============================================================================

tree_630v16c = ast.parse(
    historical_source_630v16c
)

source_lines_630v16c = (
    historical_source_630v16c.splitlines()
)

parent_map_630v16c = {}

for parent_630v16c in ast.walk(
    tree_630v16c
):
    for child_630v16c in ast.iter_child_nodes(
        parent_630v16c
    ):
        parent_map_630v16c[
            child_630v16c
        ] = parent_630v16c


# =============================================================================
# 2) IDENTIFY THE HISTORICAL FUNCTION DEFINITION RANGE
#    so calls inside the definition itself are not confused with call sites.
# =============================================================================

dispatcher_defs_630v16c = [
    node_630v16c
    for node_630v16c in ast.walk(
        tree_630v16c
    )
    if isinstance(
        node_630v16c,
        ast.FunctionDef,
    )
    and node_630v16c.name
    == TARGET_630V16C
]

if len(dispatcher_defs_630v16c) != 1:
    raise RuntimeError(
        "Historical dispatcher definition count changed."
    )

dispatcher_def_630v16c = (
    dispatcher_defs_630v16c[0]
)

dispatcher_def_range_630v16c = (
    int(
        dispatcher_def_630v16c.lineno
    ),
    int(
        dispatcher_def_630v16c.end_lineno
    ),
)


# =============================================================================
# 3) FIND TRUE CALL SITES
# =============================================================================

def _is_target_call_630v16c(
    node_630v16c,
):
    return bool(
        isinstance(
            node_630v16c,
            ast.Call,
        )
        and
        isinstance(
            node_630v16c.func,
            ast.Name,
        )
        and
        node_630v16c.func.id
        == TARGET_630V16C
    )


dispatcher_calls_630v16c = []

for node_630v16c in ast.walk(
    tree_630v16c
):
    if not _is_target_call_630v16c(
        node_630v16c
    ):
        continue

    line_630v16c = int(
        node_630v16c.lineno
    )

    # A true invocation must be outside the function's own definition range.
    if (
        dispatcher_def_range_630v16c[0]
        <= line_630v16c
        <= dispatcher_def_range_630v16c[1]
    ):
        continue

    arg_sources_630v16c = tuple(
        ast.get_source_segment(
            historical_source_630v16c,
            arg_630v16c,
        )
        for arg_630v16c
        in node_630v16c.args
    )

    keyword_sources_630v16c = tuple(
        (
            kw_630v16c.arg,
            ast.get_source_segment(
                historical_source_630v16c,
                kw_630v16c.value,
            ),
        )
        for kw_630v16c
        in node_630v16c.keywords
    )

    # Ascend to nearest enclosing statement.
    statement_630v16c = node_630v16c

    while (
        statement_630v16c
        in parent_map_630v16c
        and
        not isinstance(
            statement_630v16c,
            ast.stmt,
        )
    ):
        statement_630v16c = (
            parent_map_630v16c[
                statement_630v16c
            ]
        )

    statement_source_630v16c = (
        ast.get_source_segment(
            historical_source_630v16c,
            statement_630v16c,
        )
        if isinstance(
            statement_630v16c,
            ast.stmt,
        )
        else None
    )

    dispatcher_calls_630v16c.append(
        {
            "line":
                line_630v16c,

            "end_line":
                int(
                    getattr(
                        node_630v16c,
                        "end_lineno",
                        line_630v16c,
                    )
                ),

            "args":
                arg_sources_630v16c,

            "keywords":
                keyword_sources_630v16c,

            "statement":
                statement_source_630v16c,
        }
    )

dispatcher_calls_630v16c = tuple(
    sorted(
        dispatcher_calls_630v16c,
        key=lambda item_630v16c: (
            item_630v16c["line"],
            item_630v16c["end_line"],
        ),
    )
)


# =============================================================================
# 4) DISCOVER FIRST-ARGUMENT EXPRESSIONS EXACTLY
# =============================================================================

first_arg_sources_630v16c = tuple(
    call_630v16c["args"][0]
    for call_630v16c
    in dispatcher_calls_630v16c
    if len(
        call_630v16c["args"]
    ) >= 1
)

second_arg_sources_630v16c = tuple(
    call_630v16c["args"][1]
    for call_630v16c
    in dispatcher_calls_630v16c
    if len(
        call_630v16c["args"]
    ) >= 2
)


# =============================================================================
# 5) FIND ASSIGNMENTS TO SIMPLE FIRST-ARGUMENT NAMES
# =============================================================================

simple_first_arg_names_630v16c = tuple(
    sorted(
        {
            source_630v16c.strip()
            for source_630v16c
            in first_arg_sources_630v16c
            if source_630v16c is not None
            and source_630v16c.strip().isidentifier()
        }
    )
)

assignment_records_630v16c = []

for node_630v16c in ast.walk(
    tree_630v16c
):
    if not isinstance(
        node_630v16c,
        (ast.Assign, ast.AnnAssign),
    ):
        continue

    targets_630v16c = []

    if isinstance(
        node_630v16c,
        ast.Assign,
    ):
        targets_630v16c = list(
            node_630v16c.targets
        )
    else:
        targets_630v16c = [
            node_630v16c.target
        ]

    assigned_names_630v16c = set()

    for target_node_630v16c in targets_630v16c:
        for child_630v16c in ast.walk(
            target_node_630v16c
        ):
            if isinstance(
                child_630v16c,
                ast.Name,
            ):
                assigned_names_630v16c.add(
                    child_630v16c.id
                )

    relevant_names_630v16c = tuple(
        sorted(
            assigned_names_630v16c.intersection(
                simple_first_arg_names_630v16c
            )
        )
    )

    if not relevant_names_630v16c:
        continue

    assignment_records_630v16c.append(
        {
            "line":
                int(
                    node_630v16c.lineno
                ),

            "end_line":
                int(
                    getattr(
                        node_630v16c,
                        "end_lineno",
                        node_630v16c.lineno,
                    )
                ),

            "assigned_names":
                relevant_names_630v16c,

            "source":
                ast.get_source_segment(
                    historical_source_630v16c,
                    node_630v16c,
                ),
        }
    )

assignment_records_630v16c = tuple(
    sorted(
        assignment_records_630v16c,
        key=lambda item_630v16c: item_630v16c[
            "line"
        ],
    )
)


# =============================================================================
# 6) FIND EVERY regime_id SOURCE LINE
# =============================================================================

regime_line_numbers_630v16c = tuple(
    idx_630v16c
    for idx_630v16c, line_630v16c
    in enumerate(
        source_lines_630v16c,
        start=1,
    )
    if "regime_id" in line_630v16c
)

regime_contexts_630v16c = []

for line_no_630v16c in regime_line_numbers_630v16c:

    context_start_630v16c = max(
        1,
        line_no_630v16c - 3,
    )

    context_end_630v16c = min(
        len(
            source_lines_630v16c
        ),
        line_no_630v16c + 3,
    )

    context_text_630v16c = "\n".join(
        f"{actual_line_630v16c:04d} | "
        f"{source_lines_630v16c[actual_line_630v16c - 1]}"
        for actual_line_630v16c
        in range(
            context_start_630v16c,
            context_end_630v16c + 1,
        )
    )

    regime_contexts_630v16c.append(
        {
            "matched_line":
                int(
                    line_no_630v16c
                ),

            "context_start":
                int(
                    context_start_630v16c
                ),

            "context_end":
                int(
                    context_end_630v16c
                ),

            "context":
                context_text_630v16c,
        }
    )

regime_contexts_630v16c = tuple(
    regime_contexts_630v16c
)


# =============================================================================
# 7) PRINT LOCAL CONTEXT AROUND EACH TRUE DISPATCHER CALL
# =============================================================================

call_contexts_630v16c = []

for call_630v16c in dispatcher_calls_630v16c:

    context_start_630v16c = max(
        1,
        call_630v16c["line"] - 12,
    )

    context_end_630v16c = min(
        len(
            source_lines_630v16c
        ),
        call_630v16c["end_line"] + 12,
    )

    context_text_630v16c = "\n".join(
        f"{actual_line_630v16c:04d} | "
        f"{source_lines_630v16c[actual_line_630v16c - 1]}"
        for actual_line_630v16c
        in range(
            context_start_630v16c,
            context_end_630v16c + 1,
        )
    )

    call_contexts_630v16c.append(
        {
            "call_line":
                int(
                    call_630v16c[
                        "line"
                    ]
                ),

            "context_start":
                int(
                    context_start_630v16c
                ),

            "context_end":
                int(
                    context_end_630v16c
                ),

            "context":
                context_text_630v16c,
        }
    )

call_contexts_630v16c = tuple(
    call_contexts_630v16c
)


# =============================================================================
# 8) LOCK STATIC CONTEXT EVIDENCE
# =============================================================================

FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_AUDIT_630V16C = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16C",

    "target_function":
        TARGET_630V16C,

    "dispatcher_definition_range":
        dispatcher_def_range_630v16c,

    "true_dispatcher_call_count":
        int(
            len(
                dispatcher_calls_630v16c
            )
        ),

    "dispatcher_calls":
        dispatcher_calls_630v16c,

    "first_argument_sources":
        first_arg_sources_630v16c,

    "second_argument_sources":
        second_arg_sources_630v16c,

    "simple_first_argument_names":
        simple_first_arg_names_630v16c,

    "assignments_to_simple_first_arguments":
        assignment_records_630v16c,

    "regime_id_line_numbers":
        regime_line_numbers_630v16c,

    "regime_id_contexts":
        regime_contexts_630v16c,

    "historical_dispatcher_function_source_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B
        ),

    "historical_dispatcher_ast_audit_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_AST_AUDIT_SHA256_630V16B
        ),

    "source_executed":
        False,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_SHA256_630V16C = (
    hashlib.sha256(
        json.dumps(
            FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_AUDIT_630V16C,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_DISPATCHER_EXECUTED_630V16C = False
FD004_CORRUPTION_EXECUTED_630V16C = False
FD004_MODEL_INFERENCE_PERFORMED_630V16C = False
FD004_MC_EXECUTION_PERFORMED_630V16C = False
FD004_DEV_TARGETS_ERRORS_USED_630V16C = False
FD004_THRESHOLD_SELECTED_630V16C = False
FD004_POLICY_SELECTED_630V16C = False
FD004_TEST_CONTENTS_READ_630V16C = False
FD004_RUL_CONTENTS_READ_630V16C = False

all_checks_passed_630v16c = True


# =============================================================================
# 9) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-16C — FD004 HISTORICAL DISPATCHER "
    "CALL-SITE / base_raw + regime_id CONTEXT AUDIT"
)

print("=" * 176)

print("Historical source executed        :", False)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- TRUE HISTORICAL DISPATCHER CALLS ---")

print(
    "Call count                        :",
    len(
        dispatcher_calls_630v16c
    ),
)

for idx_630v16c, call_630v16c in enumerate(
    dispatcher_calls_630v16c,
    start=1,
):
    print(
        f"\nCALL {idx_630v16c}"
    )

    print(
        "  line                            :",
        call_630v16c["line"],
    )

    print(
        "  argument count                  :",
        len(
            call_630v16c["args"]
        ),
    )

    print(
        "  first arg / base_raw            :",
        (
            call_630v16c["args"][0]
            if len(call_630v16c["args"]) >= 1
            else None
        ),
    )

    print(
        "  second arg / scenario_row       :",
        (
            call_630v16c["args"][1]
            if len(call_630v16c["args"]) >= 2
            else None
        ),
    )

    print(
        "  enclosing statement             :"
    )

    print(
        call_630v16c["statement"]
    )

print("\n--- ASSIGNMENTS TO SIMPLE base_raw ARGUMENT NAMES ---")

if assignment_records_630v16c:
    for record_630v16c in assignment_records_630v16c:
        print(
            f"\nline {record_630v16c['line']}: "
            f"{record_630v16c['assigned_names']}"
        )

        print(
            record_630v16c[
                "source"
            ]
        )
else:
    print(
        "No simple-name first-argument assignments found."
    )

print("\n--- regime_id OCCURRENCES IN HISTORICAL CELL ---")

print(
    "regime_id matched lines           :",
    regime_line_numbers_630v16c,
)

for idx_630v16c, item_630v16c in enumerate(
    regime_contexts_630v16c,
    start=1,
):
    print(
        f"\nREGIME CONTEXT {idx_630v16c} "
        f"(match line {item_630v16c['matched_line']})"
    )

    print(
        item_630v16c[
            "context"
        ]
    )

print("\n--- LOCAL CONTEXT AROUND DISPATCHER CALLS ---")

for idx_630v16c, item_630v16c in enumerate(
    call_contexts_630v16c,
    start=1,
):
    print(
        f"\nCALL CONTEXT {idx_630v16c} "
        f"(call line {item_630v16c['call_line']})"
    )

    print(
        item_630v16c[
            "context"
        ]
    )

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Call-context audit SHA-256        :",
    FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_SHA256_630V16C,
)

print(
    "\nPASS — historical dispatcher call sites, base_raw arguments, "
    "and every regime_id occurrence were audited statically. "
    "Nothing was executed."
)

print(
    "Next: bind the FD004 dispatcher only after this output proves "
    "how historical base_raw/regime_id handling actually worked."
)

In [ ]:
# STEP 6.30V-16D — FD004 HISTORICAL FULL-NOTEBOOK
# CORRUPTION-BASE / DISPATCHER / DROPOUT USE AUDIT
#
# SOURCE ONLY. NO execution of recovered source or operators.

import hashlib
import json

required_630v16d = (
    "all_checks_passed_630v16c",
    "cells_630tr2",
    "FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B",
    "FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_SHA256_630V16C",
)

missing_630v16d = [
    n for n in required_630v16d
    if n not in globals()
]

if missing_630v16d:
    raise RuntimeError(
        "Missing STEP 6.30V-16D authority object(s): "
        + ", ".join(missing_630v16d)
    )

if not bool(all_checks_passed_630v16c):
    raise RuntimeError(
        "STEP 6.30V-16C must remain passed."
    )

TARGETS_630V16D = (
    "FD002_DEV_CORRUPTION_BASE_628GC",
    "apply_manifest_scenario_628ha",
    "apply_fd002_dropout_628gc",
    '["dropout"]',
    "regime_id",
)

def _cell_source_text_630v16d(cell):
    if not isinstance(cell, dict):
        return None

    source = cell.get("source", None)

    if isinstance(source, str):
        return source

    if isinstance(source, (list, tuple)):
        if all(isinstance(x, str) for x in source):
            return "".join(source)

    return None

cells_obj_630v16d = cells_630tr2

if isinstance(cells_obj_630v16d, dict):
    if "cells" in cells_obj_630v16d and isinstance(
        cells_obj_630v16d["cells"],
        (list, tuple),
    ):
        cell_sequence_630v16d = list(
            cells_obj_630v16d["cells"]
        )
    else:
        cell_sequence_630v16d = [
            value
            for value in cells_obj_630v16d.values()
            if isinstance(value, dict)
        ]
elif isinstance(cells_obj_630v16d, (list, tuple)):
    cell_sequence_630v16d = list(
        cells_obj_630v16d
    )
else:
    raise RuntimeError(
        "cells_630tr2 has an unsupported container type."
    )

source_cells_630v16d = []

for idx_630v16d, cell_630v16d in enumerate(
    cell_sequence_630v16d
):
    source_text_630v16d = (
        _cell_source_text_630v16d(
            cell_630v16d
        )
    )

    if source_text_630v16d is None:
        continue

    source_cells_630v16d.append(
        {
            "sequence_index":
                int(idx_630v16d),

            "cell_id":
                cell_630v16d.get(
                    "id",
                    None,
                ),

            "cell_type":
                cell_630v16d.get(
                    "cell_type",
                    None,
                ),

            "source":
                source_text_630v16d,

            "source_sha256":
                hashlib.sha256(
                    source_text_630v16d.encode(
                        "utf-8"
                    )
                ).hexdigest(),
        }
    )

if not source_cells_630v16d:
    raise RuntimeError(
        "No source-bearing historical notebook cells were recovered."
    )

occurrences_630v16d = {
    target_630v16d: []
    for target_630v16d in TARGETS_630V16D
}

for cell_info_630v16d in source_cells_630v16d:

    lines_630v16d = (
        cell_info_630v16d[
            "source"
        ].splitlines()
    )

    for target_630v16d in TARGETS_630V16D:

        matched_lines_630v16d = [
            line_no_630v16d
            for line_no_630v16d, line_text_630v16d
            in enumerate(
                lines_630v16d,
                start=1,
            )
            if target_630v16d
            in line_text_630v16d
        ]

        for matched_line_630v16d in matched_lines_630v16d:

            context_start_630v16d = max(
                1,
                matched_line_630v16d - 8,
            )

            context_end_630v16d = min(
                len(lines_630v16d),
                matched_line_630v16d + 12,
            )

            context_630v16d = "\n".join(
                f"{line_no_630v16d:04d} | "
                f"{lines_630v16d[line_no_630v16d - 1]}"
                for line_no_630v16d
                in range(
                    context_start_630v16d,
                    context_end_630v16d + 1,
                )
            )

            occurrences_630v16d[
                target_630v16d
            ].append(
                {
                    "sequence_index":
                        cell_info_630v16d[
                            "sequence_index"
                        ],

                    "cell_id":
                        cell_info_630v16d[
                            "cell_id"
                        ],

                    "cell_type":
                        cell_info_630v16d[
                            "cell_type"
                        ],

                    "cell_source_sha256":
                        cell_info_630v16d[
                            "source_sha256"
                        ],

                    "matched_line":
                        int(
                            matched_line_630v16d
                        ),

                    "context_start":
                        int(
                            context_start_630v16d
                        ),

                    "context_end":
                        int(
                            context_end_630v16d
                        ),

                    "context":
                        context_630v16d,
                }
            )

dispatcher_call_occurrences_630v16d = []

for cell_info_630v16d in source_cells_630v16d:

    lines_630v16d = (
        cell_info_630v16d[
            "source"
        ].splitlines()
    )

    for line_no_630v16d, line_text_630v16d in enumerate(
        lines_630v16d,
        start=1,
    ):
        stripped_630v16d = (
            line_text_630v16d.strip()
        )

        if (
            "apply_manifest_scenario_628ha("
            in stripped_630v16d
            and
            not stripped_630v16d.startswith(
                "def apply_manifest_scenario_628ha"
            )
        ):
            context_start_630v16d = max(
                1,
                line_no_630v16d - 10,
            )
            context_end_630v16d = min(
                len(lines_630v16d),
                line_no_630v16d + 12,
            )

            dispatcher_call_occurrences_630v16d.append(
                {
                    "sequence_index":
                        cell_info_630v16d[
                            "sequence_index"
                        ],
                    "cell_id":
                        cell_info_630v16d[
                            "cell_id"
                        ],
                    "cell_source_sha256":
                        cell_info_630v16d[
                            "source_sha256"
                        ],
                    "matched_line":
                        int(
                            line_no_630v16d
                        ),
                    "context":
                        "\n".join(
                            f"{n_630v16d:04d} | "
                            f"{lines_630v16d[n_630v16d - 1]}"
                            for n_630v16d
                            in range(
                                context_start_630v16d,
                                context_end_630v16d + 1,
                            )
                        ),
                }
            )

resident_base_present_630v16d = bool(
    "FD002_DEV_CORRUPTION_BASE_628GC"
    in globals()
)

resident_base_metadata_630v16d = None

if resident_base_present_630v16d:
    base_obj_630v16d = globals()[
        "FD002_DEV_CORRUPTION_BASE_628GC"
    ]

    resident_base_metadata_630v16d = {
        "type":
            type(
                base_obj_630v16d
            ).__name__,

        "shape":
            (
                tuple(
                    base_obj_630v16d.shape
                )
                if hasattr(
                    base_obj_630v16d,
                    "shape",
                )
                else None
            ),

        "columns":
            (
                tuple(
                    str(c)
                    for c
                    in base_obj_630v16d.columns
                )
                if hasattr(
                    base_obj_630v16d,
                    "columns",
                )
                else None
            ),

        "has_regime_id":
            (
                bool(
                    "regime_id"
                    in base_obj_630v16d.columns
                )
                if hasattr(
                    base_obj_630v16d,
                    "columns",
                )
                else None
            ),
    }

FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_AUDIT_630V16D = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16D",

    "source_cell_count":
        int(
            len(
                source_cells_630v16d
            )
        ),

    "target_occurrence_counts":
        {
            target_630v16d:
                int(
                    len(
                        occurrences_630v16d[
                            target_630v16d
                        ]
                    )
                )
            for target_630v16d
            in TARGETS_630V16D
        },

    "dispatcher_call_count_across_recovered_notebook":
        int(
            len(
                dispatcher_call_occurrences_630v16d
            )
        ),

    "resident_historical_base_present":
        resident_base_present_630v16d,

    "resident_historical_base_metadata":
        resident_base_metadata_630v16d,

    "historical_dispatcher_function_source_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B
        ),

    "historical_dispatcher_call_context_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_CALL_CONTEXT_SHA256_630V16C
        ),

    "source_executed":
        False,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D = (
    hashlib.sha256(
        json.dumps(
            FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_AUDIT_630V16D,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_DISPATCHER_EXECUTED_630V16D = False
FD004_CORRUPTION_EXECUTED_630V16D = False
FD004_MODEL_INFERENCE_PERFORMED_630V16D = False
FD004_MC_EXECUTION_PERFORMED_630V16D = False
FD004_DEV_TARGETS_ERRORS_USED_630V16D = False
FD004_THRESHOLD_SELECTED_630V16D = False
FD004_POLICY_SELECTED_630V16D = False
FD004_TEST_CONTENTS_READ_630V16D = False
FD004_RUL_CONTENTS_READ_630V16D = False

all_checks_passed_630v16d = True

print("=" * 176)

print(
    "STEP 6.30V-16D — FD004 HISTORICAL FULL-NOTEBOOK "
    "CORRUPTION-BASE / DISPATCHER / DROPOUT USE AUDIT"
)

print("=" * 176)

print("Historical source executed        :", False)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- RECOVERED NOTEBOOK COVERAGE ---")

print(
    "Source-bearing cells              :",
    len(
        source_cells_630v16d
    ),
)

print(
    "Dispatcher calls across notebook  :",
    len(
        dispatcher_call_occurrences_630v16d
    ),
)

print("\n--- TARGET OCCURRENCE COUNTS ---")

for target_630v16d in TARGETS_630V16D:
    print(
        f"{target_630v16d:40s}:",
        len(
            occurrences_630v16d[
                target_630v16d
            ]
        ),
    )

print("\n--- FD002_DEV_CORRUPTION_BASE_628GC CONTEXTS ---")

for idx_630v16d, item_630v16d in enumerate(
    occurrences_630v16d[
        "FD002_DEV_CORRUPTION_BASE_628GC"
    ],
    start=1,
):
    print(
        f"\nBASE CONTEXT {idx_630v16d}"
    )

    print(
        "cell sequence index               :",
        item_630v16d[
            "sequence_index"
        ],
    )

    print(
        "cell source SHA-256               :",
        item_630v16d[
            "cell_source_sha256"
        ],
    )

    print(
        item_630v16d[
            "context"
        ],
    )

print("\n--- ALL HISTORICAL DISPATCHER CALL CONTEXTS ---")

for idx_630v16d, item_630v16d in enumerate(
    dispatcher_call_occurrences_630v16d,
    start=1,
):
    print(
        f"\nDISPATCH CALL {idx_630v16d}"
    )

    print(
        "cell sequence index               :",
        item_630v16d[
            "sequence_index"
        ],
    )

    print(
        "cell source SHA-256               :",
        item_630v16d[
            "cell_source_sha256"
        ],
    )

    print(
        item_630v16d[
            "context"
        ],
    )

print("\n--- HISTORICAL DROPOUT FUNCTION/CONTRACT CONTEXTS ---")

dropout_context_items_630v16d = (
    list(
        occurrences_630v16d[
            "apply_fd002_dropout_628gc"
        ]
    )
    +
    list(
        occurrences_630v16d[
            '["dropout"]'
        ]
    )
)

for idx_630v16d, item_630v16d in enumerate(
    dropout_context_items_630v16d,
    start=1,
):
    print(
        f"\nDROPOUT CONTEXT {idx_630v16d}"
    )

    print(
        "cell sequence index               :",
        item_630v16d[
            "sequence_index"
        ],
    )

    print(
        "cell source SHA-256               :",
        item_630v16d[
            "cell_source_sha256"
        ],
    )

    print(
        item_630v16d[
            "context"
        ],
    )

print("\n--- RESIDENT HISTORICAL BASE METADATA ---")

print(
    "Resident historical base present :",
    resident_base_present_630v16d,
)

print(
    "Resident base metadata            :",
    resident_base_metadata_630v16d,
)

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Base/context audit SHA-256        :",
    FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D,
)

print(
    "\nPASS — the recovered notebook was searched for the exact "
    "historical corruption-base definition, every dispatcher call, "
    "and dropout-use context without executing anything."
)

print(
    "Next: bind the FD004 dispatcher only from what these historical "
    "contexts actually prove."
)

In [ ]:
# STEP 6.30V-16E — FD004 EXACT HISTORICAL FULL-RUN DISPATCHER
# apply_scenario_628hc0 SOURCE RECOVERY + STATIC AUDIT
#
# SOURCE ONLY. Nothing recovered here is executed.

import ast
import hashlib
import json

required_630v16e = (
    "all_checks_passed_630v16d",
    "cells_630tr2",
    "FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D",
)

missing_630v16e = [
    n for n in required_630v16e
    if n not in globals()
]

if missing_630v16e:
    raise RuntimeError(
        "Missing STEP 6.30V-16E authority object(s): "
        + ", ".join(missing_630v16e)
    )

if not bool(all_checks_passed_630v16d):
    raise RuntimeError(
        "STEP 6.30V-16D must remain passed."
    )

TARGET_630V16E = "apply_scenario_628hc0"


def _cell_source_630v16e(cell_630v16e):
    if not isinstance(cell_630v16e, dict):
        return None

    src_630v16e = cell_630v16e.get("source", None)

    if isinstance(src_630v16e, str):
        return src_630v16e

    if isinstance(src_630v16e, (list, tuple)):
        if all(isinstance(x, str) for x in src_630v16e):
            return "".join(src_630v16e)

    return None


cells_obj_630v16e = cells_630tr2

if isinstance(cells_obj_630v16e, dict):
    if (
        "cells" in cells_obj_630v16e
        and isinstance(
            cells_obj_630v16e["cells"],
            (list, tuple),
        )
    ):
        cell_sequence_630v16e = list(
            cells_obj_630v16e["cells"]
        )
    else:
        cell_sequence_630v16e = [
            value_630v16e
            for value_630v16e
            in cells_obj_630v16e.values()
            if isinstance(value_630v16e, dict)
        ]

elif isinstance(cells_obj_630v16e, (list, tuple)):
    cell_sequence_630v16e = list(cells_obj_630v16e)

else:
    raise RuntimeError(
        "cells_630tr2 has unsupported container type."
    )


definition_hits_630v16e = []

for cell_index_630v16e, cell_630v16e in enumerate(
    cell_sequence_630v16e
):
    source_630v16e = _cell_source_630v16e(
        cell_630v16e
    )

    if source_630v16e is None:
        continue

    if TARGET_630V16E not in source_630v16e:
        continue

    try:
        tree_630v16e = ast.parse(source_630v16e)
    except SyntaxError:
        continue

    for node_630v16e in ast.walk(tree_630v16e):
        if (
            isinstance(node_630v16e, ast.FunctionDef)
            and node_630v16e.name == TARGET_630V16E
        ):
            if not hasattr(node_630v16e, "end_lineno"):
                raise RuntimeError(
                    "AST end_lineno unavailable for exact source extraction."
                )

            lines_630v16e = source_630v16e.splitlines(
                keepends=True
            )

            fn_source_630v16e = "".join(
                lines_630v16e[
                    node_630v16e.lineno - 1:
                    node_630v16e.end_lineno
                ]
            )

            definition_hits_630v16e.append(
                {
                    "cell_sequence_index":
                        int(cell_index_630v16e),

                    "cell_id":
                        cell_630v16e.get(
                            "id",
                            None,
                        ),

                    "cell_source_sha256":
                        hashlib.sha256(
                            source_630v16e.encode("utf-8")
                        ).hexdigest(),

                    "function_start_line":
                        int(node_630v16e.lineno),

                    "function_end_line":
                        int(node_630v16e.end_lineno),

                    "function_source":
                        fn_source_630v16e,

                    "function_source_sha256":
                        hashlib.sha256(
                            fn_source_630v16e.encode("utf-8")
                        ).hexdigest(),

                    "node":
                        node_630v16e,

                    "cell_source":
                        source_630v16e,
                }
            )


if len(definition_hits_630v16e) != 1:
    raise RuntimeError(
        "Expected exactly one historical full-run dispatcher definition "
        f"for {TARGET_630V16E}; found {len(definition_hits_630v16e)}."
    )

hit_630v16e = definition_hits_630v16e[0]

node_630v16e = hit_630v16e["node"]
source_630v16e = hit_630v16e["cell_source"]
fn_source_630v16e = hit_630v16e["function_source"]


# ------------------------------------------------------------------
# Static signature
# ------------------------------------------------------------------

args_630v16e = node_630v16e.args

positional_args_630v16e = tuple(
    arg_630v16e.arg
    for arg_630v16e
    in (
        list(args_630v16e.posonlyargs)
        + list(args_630v16e.args)
    )
)

defaults_630v16e = tuple(
    ast.get_source_segment(
        source_630v16e,
        default_630v16e,
    )
    for default_630v16e
    in args_630v16e.defaults
)

kwonly_args_630v16e = tuple(
    arg_630v16e.arg
    for arg_630v16e
    in args_630v16e.kwonlyargs
)

vararg_630v16e = (
    None
    if args_630v16e.vararg is None
    else args_630v16e.vararg.arg
)

kwarg_630v16e = (
    None
    if args_630v16e.kwarg is None
    else args_630v16e.kwarg.arg
)


# ------------------------------------------------------------------
# Static branches
# ------------------------------------------------------------------

branch_records_630v16e = []

for child_630v16e in ast.walk(node_630v16e):
    if isinstance(child_630v16e, ast.If):
        branch_records_630v16e.append(
            {
                "line":
                    int(child_630v16e.lineno),

                "test":
                    ast.get_source_segment(
                        source_630v16e,
                        child_630v16e.test,
                    ),
            }
        )

branch_records_630v16e = tuple(
    sorted(
        branch_records_630v16e,
        key=lambda x_630v16e: x_630v16e["line"],
    )
)


# ------------------------------------------------------------------
# Static calls
# ------------------------------------------------------------------

def _call_name_630v16e(call_630v16e):
    fn_630v16e = call_630v16e.func

    if isinstance(fn_630v16e, ast.Name):
        return fn_630v16e.id

    if isinstance(fn_630v16e, ast.Attribute):
        parts_630v16e = []
        cur_630v16e = fn_630v16e

        while isinstance(cur_630v16e, ast.Attribute):
            parts_630v16e.append(cur_630v16e.attr)
            cur_630v16e = cur_630v16e.value

        if isinstance(cur_630v16e, ast.Name):
            parts_630v16e.append(cur_630v16e.id)

        return ".".join(reversed(parts_630v16e))

    return ast.dump(
        fn_630v16e,
        include_attributes=False,
    )


call_records_630v16e = []

for child_630v16e in ast.walk(node_630v16e):
    if isinstance(child_630v16e, ast.Call):
        call_records_630v16e.append(
            {
                "line":
                    int(child_630v16e.lineno),

                "call_name":
                    _call_name_630v16e(
                        child_630v16e
                    ),

                "source":
                    ast.get_source_segment(
                        source_630v16e,
                        child_630v16e,
                    ),
            }
        )

call_records_630v16e = tuple(
    sorted(
        call_records_630v16e,
        key=lambda x_630v16e: (
            x_630v16e["line"],
            x_630v16e["call_name"],
        ),
    )
)


# ------------------------------------------------------------------
# Static assignments and return expressions
# ------------------------------------------------------------------

assignment_records_630v16e = []

for child_630v16e in ast.walk(node_630v16e):
    if isinstance(child_630v16e, ast.Assign):
        assignment_records_630v16e.append(
            {
                "line":
                    int(child_630v16e.lineno),

                "source":
                    ast.get_source_segment(
                        source_630v16e,
                        child_630v16e,
                    ),
            }
        )

assignment_records_630v16e = tuple(
    sorted(
        assignment_records_630v16e,
        key=lambda x_630v16e: x_630v16e["line"],
    )
)


return_records_630v16e = []

for child_630v16e in ast.walk(node_630v16e):
    if isinstance(child_630v16e, ast.Return):
        return_records_630v16e.append(
            {
                "line":
                    int(child_630v16e.lineno),

                "source":
                    ast.get_source_segment(
                        source_630v16e,
                        child_630v16e,
                    ),
            }
        )

return_records_630v16e = tuple(
    sorted(
        return_records_630v16e,
        key=lambda x_630v16e: x_630v16e["line"],
    )
)


# ------------------------------------------------------------------
# Lock source evidence
# ------------------------------------------------------------------

FD004_HISTORICAL_FULLRUN_DISPATCHER_SOURCE_630V16E = (
    fn_source_630v16e
)

FD004_HISTORICAL_FULLRUN_DISPATCHER_SOURCE_SHA256_630V16E = (
    hit_630v16e[
        "function_source_sha256"
    ]
)

FD004_HISTORICAL_FULLRUN_DISPATCHER_AUDIT_630V16E = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16E",

    "target_function":
        TARGET_630V16E,

    "definition_count":
        1,

    "cell_sequence_index":
        hit_630v16e[
            "cell_sequence_index"
        ],

    "cell_source_sha256":
        hit_630v16e[
            "cell_source_sha256"
        ],

    "function_start_line":
        hit_630v16e[
            "function_start_line"
        ],

    "function_end_line":
        hit_630v16e[
            "function_end_line"
        ],

    "function_source_sha256":
        hit_630v16e[
            "function_source_sha256"
        ],

    "positional_args":
        positional_args_630v16e,

    "defaults":
        defaults_630v16e,

    "keyword_only_args":
        kwonly_args_630v16e,

    "vararg":
        vararg_630v16e,

    "kwarg":
        kwarg_630v16e,

    "branches":
        branch_records_630v16e,

    "calls":
        call_records_630v16e,

    "assignments":
        assignment_records_630v16e,

    "returns":
        return_records_630v16e,

    "historical_base_context_sha256":
        str(
            FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D
        ),

    "source_executed":
        False,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_HISTORICAL_FULLRUN_DISPATCHER_AUDIT_SHA256_630V16E = (
    hashlib.sha256(
        json.dumps(
            FD004_HISTORICAL_FULLRUN_DISPATCHER_AUDIT_630V16E,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_FULLRUN_DISPATCHER_SOURCE_EXECUTED_630V16E = False
FD004_FULLRUN_DISPATCHER_EXECUTED_630V16E = False
FD004_CORRUPTION_EXECUTED_630V16E = False
FD004_MODEL_INFERENCE_PERFORMED_630V16E = False
FD004_MC_EXECUTION_PERFORMED_630V16E = False
FD004_DEV_TARGETS_ERRORS_USED_630V16E = False
FD004_THRESHOLD_SELECTED_630V16E = False
FD004_POLICY_SELECTED_630V16E = False
FD004_TEST_CONTENTS_READ_630V16E = False
FD004_RUL_CONTENTS_READ_630V16E = False

all_checks_passed_630v16e = True


# =============================================================================
# REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-16E — FD004 EXACT HISTORICAL FULL-RUN DISPATCHER "
    "SOURCE RECOVERY + STATIC AUDIT"
)

print("=" * 176)

print("Recovered source executed         :", False)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- EXACT DEFINITION ---")

print(
    "Target function                   :",
    TARGET_630V16E,
)

print(
    "Definition count                  :",
    len(
        definition_hits_630v16e
    ),
)

print(
    "Historical cell sequence index    :",
    hit_630v16e[
        "cell_sequence_index"
    ],
)

print(
    "Historical cell SHA-256           :",
    hit_630v16e[
        "cell_source_sha256"
    ],
)

print(
    "Function line range               :",
    (
        hit_630v16e[
            "function_start_line"
        ],
        hit_630v16e[
            "function_end_line"
        ],
    ),
)

print(
    "Function source SHA-256           :",
    FD004_HISTORICAL_FULLRUN_DISPATCHER_SOURCE_SHA256_630V16E,
)

print("\n--- STATIC SIGNATURE ---")

print(
    "Positional args                   :",
    positional_args_630v16e,
)

print(
    "Defaults                          :",
    defaults_630v16e,
)

print(
    "Keyword-only args                 :",
    kwonly_args_630v16e,
)

print(
    "*args                             :",
    vararg_630v16e,
)

print(
    "**kwargs                          :",
    kwarg_630v16e,
)

print("\n--- BRANCH TESTS ---")

for record_630v16e in branch_records_630v16e:
    print(
        f"line {record_630v16e['line']:4d}: "
        f"{record_630v16e['test']}"
    )

print("\n--- ASSIGNMENTS ---")

for record_630v16e in assignment_records_630v16e:
    print(
        f"\nline {record_630v16e['line']:4d}:"
    )
    print(
        record_630v16e[
            "source"
        ]
    )

print("\n--- CALL EXPRESSIONS ---")

for record_630v16e in call_records_630v16e:
    print(
        f"\nline {record_630v16e['line']:4d}: "
        f"{record_630v16e['call_name']}"
    )
    print(
        record_630v16e[
            "source"
        ]
    )

print("\n--- RETURN EXPRESSIONS ---")

for record_630v16e in return_records_630v16e:
    print(
        f"line {record_630v16e['line']:4d}: "
        f"{record_630v16e['source']}"
    )

print("\n--- EXACT HISTORICAL FULL-RUN DISPATCHER SOURCE ---")

for line_no_630v16e, line_text_630v16e in enumerate(
    fn_source_630v16e.splitlines(),
    start=1,
):
    print(
        f"{line_no_630v16e:03d} | "
        f"{line_text_630v16e}"
    )

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Full-run dispatcher audit SHA-256 :",
    FD004_HISTORICAL_FULLRUN_DISPATCHER_AUDIT_SHA256_630V16E,
)

print(
    "\nPASS — the exact dispatcher used by the historical full "
    "corruption workflow was recovered and audited statically. "
    "It was not executed."
)

print(
    "Next: construct the FD004 corruption base with its proven "
    "regime_id column and bind the FD004 full-run dispatcher strictly "
    "from this exact historical source."
)

In [ ]:
# STEP 6.30V-16E-R1 — FD004 HISTORICAL FULL-RUN HELPER
# ALIAS + BRIDGE-CONTRACT ASSIGNMENT AUDIT
#
# SOURCE ONLY.
# NO recovered source execution.
# NO dispatcher execution.
# NO corruption.
# NO preprocessing / model / MC.
# NO outcomes / threshold / TEST / RUL.

import ast
import hashlib
import json

required_630v16er1 = (
    "all_checks_passed_630v16d",
    "cells_630tr2",
    "FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D",
)

missing_630v16er1 = [
    n for n in required_630v16er1
    if n not in globals()
]

if missing_630v16er1:
    raise RuntimeError(
        "Missing STEP 6.30V-16E-R1 authority object(s): "
        + ", ".join(missing_630v16er1)
    )

if not bool(all_checks_passed_630v16d):
    raise RuntimeError(
        "STEP 6.30V-16D must remain passed."
    )

TARGET_ALIAS_630V16ER1 = "apply_scenario_628hc0"
TARGET_CONTRACT_630V16ER1 = (
    "FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA"
)


def _cell_source_630v16er1(cell):
    if not isinstance(cell, dict):
        return None

    src = cell.get("source", None)

    if isinstance(src, str):
        return src

    if isinstance(src, (list, tuple)):
        if all(isinstance(x, str) for x in src):
            return "".join(src)

    return None


cells_obj_630v16er1 = cells_630tr2

if isinstance(cells_obj_630v16er1, dict):
    if (
        "cells" in cells_obj_630v16er1
        and isinstance(
            cells_obj_630v16er1["cells"],
            (list, tuple),
        )
    ):
        cell_sequence_630v16er1 = list(
            cells_obj_630v16er1["cells"]
        )
    else:
        cell_sequence_630v16er1 = [
            value
            for value in cells_obj_630v16er1.values()
            if isinstance(value, dict)
        ]

elif isinstance(cells_obj_630v16er1, (list, tuple)):
    cell_sequence_630v16er1 = list(
        cells_obj_630v16er1
    )

else:
    raise RuntimeError(
        "cells_630tr2 has unsupported container type."
    )


def _assigned_names_630v16er1(assign_node):
    names = []

    if isinstance(assign_node, ast.Assign):
        targets = assign_node.targets

    elif isinstance(assign_node, ast.AnnAssign):
        targets = [assign_node.target]

    else:
        return tuple()

    for target in targets:

        if isinstance(target, ast.Name):
            names.append(
                target.id
            )

        elif isinstance(
            target,
            (ast.Tuple, ast.List),
        ):
            for child in target.elts:
                if isinstance(child, ast.Name):
                    names.append(
                        child.id
                    )

    return tuple(names)


assignment_hits_630v16er1 = []

for cell_index, cell in enumerate(
    cell_sequence_630v16er1
):
    source = _cell_source_630v16er1(
        cell
    )

    if source is None:
        continue

    if (
        TARGET_ALIAS_630V16ER1 not in source
        and
        TARGET_CONTRACT_630V16ER1 not in source
    ):
        continue

    try:
        tree = ast.parse(source)

    except SyntaxError:
        continue

    for node in ast.walk(tree):

        if not isinstance(
            node,
            (ast.Assign, ast.AnnAssign),
        ):
            continue

        assigned = (
            _assigned_names_630v16er1(
                node
            )
        )

        relevant = tuple(
            name
            for name in assigned
            if name in (
                TARGET_ALIAS_630V16ER1,
                TARGET_CONTRACT_630V16ER1,
            )
        )

        if not relevant:
            continue

        assignment_source = (
            ast.get_source_segment(
                source,
                node,
            )
        )

        rhs_node = node.value

        rhs_source = (
            None
            if rhs_node is None
            else ast.get_source_segment(
                source,
                rhs_node,
            )
        )

        assignment_hits_630v16er1.append(
            {
                "cell_sequence_index":
                    int(cell_index),

                "cell_id":
                    cell.get(
                        "id",
                        None,
                    ),

                "cell_source_sha256":
                    hashlib.sha256(
                        source.encode("utf-8")
                    ).hexdigest(),

                "line":
                    int(node.lineno),

                "end_line":
                    int(
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        )
                    ),

                "assigned_names":
                    relevant,

                "assignment_source":
                    assignment_source,

                "rhs_source":
                    rhs_source,
            }
        )


alias_hits_630v16er1 = [
    item
    for item in assignment_hits_630v16er1
    if TARGET_ALIAS_630V16ER1
    in item["assigned_names"]
]

contract_hits_630v16er1 = [
    item
    for item in assignment_hits_630v16er1
    if TARGET_CONTRACT_630V16ER1
    in item["assigned_names"]
]


if len(alias_hits_630v16er1) != 1:
    raise RuntimeError(
        "Expected exactly one assignment to "
        f"{TARGET_ALIAS_630V16ER1}; found "
        f"{len(alias_hits_630v16er1)}."
    )

if len(contract_hits_630v16er1) != 1:
    raise RuntimeError(
        "Expected exactly one assignment to "
        f"{TARGET_CONTRACT_630V16ER1}; found "
        f"{len(contract_hits_630v16er1)}."
    )


alias_hit_630v16er1 = (
    alias_hits_630v16er1[0]
)

contract_hit_630v16er1 = (
    contract_hits_630v16er1[0]
)


FD004_HISTORICAL_FULLRUN_ALIAS_ASSIGNMENT_630V16ER1 = (
    alias_hit_630v16er1[
        "assignment_source"
    ]
)

FD004_HISTORICAL_BRIDGE_CONTRACT_ASSIGNMENT_630V16ER1 = (
    contract_hit_630v16er1[
        "assignment_source"
    ]
)

FD004_HISTORICAL_FULLRUN_ALIAS_RHS_630V16ER1 = (
    alias_hit_630v16er1[
        "rhs_source"
    ]
)

FD004_HISTORICAL_BRIDGE_CONTRACT_RHS_630V16ER1 = (
    contract_hit_630v16er1[
        "rhs_source"
    ]
)


alias_assignment_sha_630v16er1 = (
    hashlib.sha256(
        FD004_HISTORICAL_FULLRUN_ALIAS_ASSIGNMENT_630V16ER1
        .encode("utf-8")
    ).hexdigest()
)

contract_assignment_sha_630v16er1 = (
    hashlib.sha256(
        FD004_HISTORICAL_BRIDGE_CONTRACT_ASSIGNMENT_630V16ER1
        .encode("utf-8")
    ).hexdigest()
)


FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_630V16ER1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-16E-R1",

    "alias_name":
        TARGET_ALIAS_630V16ER1,

    "bridge_contract_name":
        TARGET_CONTRACT_630V16ER1,

    "alias_assignment_count":
        int(
            len(
                alias_hits_630v16er1
            )
        ),

    "bridge_contract_assignment_count":
        int(
            len(
                contract_hits_630v16er1
            )
        ),

    "alias_assignment_cell_index":
        alias_hit_630v16er1[
            "cell_sequence_index"
        ],

    "alias_assignment_cell_sha256":
        alias_hit_630v16er1[
            "cell_source_sha256"
        ],

    "alias_assignment_line":
        alias_hit_630v16er1[
            "line"
        ],

    "alias_assignment_rhs":
        FD004_HISTORICAL_FULLRUN_ALIAS_RHS_630V16ER1,

    "alias_assignment_sha256":
        alias_assignment_sha_630v16er1,

    "bridge_contract_cell_index":
        contract_hit_630v16er1[
            "cell_sequence_index"
        ],

    "bridge_contract_cell_sha256":
        contract_hit_630v16er1[
            "cell_source_sha256"
        ],

    "bridge_contract_assignment_line":
        contract_hit_630v16er1[
            "line"
        ],

    "bridge_contract_rhs":
        FD004_HISTORICAL_BRIDGE_CONTRACT_RHS_630V16ER1,

    "bridge_contract_assignment_sha256":
        contract_assignment_sha_630v16er1,

    "historical_base_context_sha256":
        str(
            FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D
        ),

    "source_executed":
        False,

    "alias_resolved_at_runtime":
        False,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1 = (
    hashlib.sha256(
        json.dumps(
            FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_630V16ER1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DISPATCHER_EXECUTED_630V16ER1 = False
FD004_CORRUPTION_EXECUTED_630V16ER1 = False
FD004_MODEL_INFERENCE_PERFORMED_630V16ER1 = False
FD004_MC_EXECUTION_PERFORMED_630V16ER1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V16ER1 = False
FD004_THRESHOLD_SELECTED_630V16ER1 = False
FD004_POLICY_SELECTED_630V16ER1 = False
FD004_TEST_CONTENTS_READ_630V16ER1 = False
FD004_RUL_CONTENTS_READ_630V16ER1 = False

all_checks_passed_630v16er1 = True


print("=" * 176)

print(
    "STEP 6.30V-16E-R1 — FD004 HISTORICAL FULL-RUN HELPER "
    "ALIAS + BRIDGE-CONTRACT ASSIGNMENT AUDIT"
)

print("=" * 176)

print("Recovered source executed         :", False)
print("Alias resolved at runtime         :", False)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- apply_scenario_628hc0 ASSIGNMENT ---")

print(
    "Assignment count                  :",
    len(
        alias_hits_630v16er1
    ),
)

print(
    "Cell sequence index               :",
    alias_hit_630v16er1[
        "cell_sequence_index"
    ],
)

print(
    "Cell source SHA-256               :",
    alias_hit_630v16er1[
        "cell_source_sha256"
    ],
)

print(
    "Assignment line                   :",
    alias_hit_630v16er1[
        "line"
    ],
)

print("Exact assignment:")

print(
    FD004_HISTORICAL_FULLRUN_ALIAS_ASSIGNMENT_630V16ER1
)

print("\nRHS only:")

print(
    FD004_HISTORICAL_FULLRUN_ALIAS_RHS_630V16ER1
)

print(
    "\n--- FD002_CORRUPTION_MODEL_BRIDGE_CONTRACT_628HA "
    "ASSIGNMENT ---"
)

print(
    "Assignment count                  :",
    len(
        contract_hits_630v16er1
    ),
)

print(
    "Cell sequence index               :",
    contract_hit_630v16er1[
        "cell_sequence_index"
    ],
)

print(
    "Cell source SHA-256               :",
    contract_hit_630v16er1[
        "cell_source_sha256"
    ],
)

print(
    "Assignment line                   :",
    contract_hit_630v16er1[
        "line"
    ],
)

print("Exact assignment:")

print(
    FD004_HISTORICAL_BRIDGE_CONTRACT_ASSIGNMENT_630V16ER1
)

print("\n--- FINGERPRINTS ---")

print(
    "Alias assignment SHA-256          :",
    alias_assignment_sha_630v16er1,
)

print(
    "Bridge contract assignment SHA    :",
    contract_assignment_sha_630v16er1,
)

print(
    "Alias/contract audit SHA-256      :",
    FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1,
)

print(
    "\nPASS — the historical full-run helper was audited as an "
    "assignment/alias rather than incorrectly assumed to be a "
    "def-defined function. Nothing was executed."
)

print(
    "Next: interpret only these exact printed assignments, then bind "
    "the FD004 corruption base and dispatcher without guessing."
)

In [ ]:
# STEP 6.30V-17A — FD004 CORRUPTION BASE +
# EXACT HISTORICAL-ROUTING DISPATCHER BIND
#
# BINDING ONLY.
# NO corruption scenario is executed.
# NO preprocessing / model / MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import pandas as pd
import hashlib
import json
import inspect
from types import MappingProxyType


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v17a = (
    "all_checks_passed_630v16er1",
    "all_checks_passed_630v15",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_CANONICAL_REGIME_CENTERS_630F",
    "FD004_CORRUPTION_OPERATORS_630V13",
    "FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13",
    "FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15",
    "FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B",
    "FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D",
    "FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1",
)

missing_630v17a = [
    name_630v17a
    for name_630v17a in required_630v17a
    if name_630v17a not in globals()
]

if missing_630v17a:
    raise RuntimeError(
        "Missing STEP 6.30V-17A authority object(s): "
        + ", ".join(missing_630v17a)
    )

if not (
    bool(all_checks_passed_630v16er1)
    and bool(all_checks_passed_630v15)
):
    raise RuntimeError(
        "Required prior authority stages are not passed."
    )


# =============================================================================
# 1) EXACT HISTORICAL AUTHORITY FINGERPRINT GUARDS
# =============================================================================

EXPECTED_HISTORICAL_DISPATCHER_FUNCTION_SHA256_630V17A = (
    "f4813865d16ec633ff8ea90ea5aa96bd985440c3d0220c6d9d85b7c8e166100d"
)

EXPECTED_HISTORICAL_BASE_CONTEXT_SHA256_630V17A = (
    "02d726c2ba50b097aa71b1bb6bd6f12f3c181d4dcb59aa5edc0a62c5ee72b20b"
)

EXPECTED_HISTORICAL_ALIAS_AUDIT_SHA256_630V17A = (
    "cd6a18c0c436109c789910ace6ca3ef92c653ec4e94bd162be0869b02e040dc4"
)

if (
    str(FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B)
    != EXPECTED_HISTORICAL_DISPATCHER_FUNCTION_SHA256_630V17A
):
    raise RuntimeError(
        "Historical dispatcher-function fingerprint changed."
    )

if (
    str(FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D)
    != EXPECTED_HISTORICAL_BASE_CONTEXT_SHA256_630V17A
):
    raise RuntimeError(
        "Historical corruption-base context fingerprint changed."
    )

if (
    str(FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1)
    != EXPECTED_HISTORICAL_ALIAS_AUDIT_SHA256_630V17A
):
    raise RuntimeError(
        "Historical full-run alias audit fingerprint changed."
    )


# =============================================================================
# 2) SNAPSHOT FD004 AUTHORITIES BEFORE CONSTRUCTION
# =============================================================================

raw_authority_630v17a = FD004_DEV_RAW_630C

raw_snapshot_630v17a = (
    raw_authority_630v17a.copy(deep=True)
)

regime_ids_630v17a = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

regime_snapshot_630v17a = (
    regime_ids_630v17a.copy()
)

if regime_ids_630v17a.shape != (
    len(raw_authority_630v17a),
):
    raise RuntimeError(
        "FD004 DEV raw and frozen regime-ID lengths differ."
    )

n_regimes_630v17a = int(
    np.asarray(
        FD004_CANONICAL_REGIME_CENTERS_630F
    ).shape[0]
)

if n_regimes_630v17a <= 0:
    raise RuntimeError(
        "Canonical regime authority is invalid."
    )

if not np.all(
    (regime_ids_630v17a >= 0)
    &
    (regime_ids_630v17a < n_regimes_630v17a)
):
    raise RuntimeError(
        "DEV regime IDs fall outside the frozen canonical range."
    )


# =============================================================================
# 3) BUILD FD004 CORRUPTION BASE IN THE PROVEN HISTORICAL FORM
#
# Historical authority:
# raw DEV copy -> append aligned regime_id -> reset-index use in execution.
#
# We construct the final execution base directly as:
# raw DEV deep copy -> reset_index(drop=True) -> append regime_id.
# =============================================================================

if "regime_id" in raw_authority_630v17a.columns:
    raise RuntimeError(
        "FD004 raw DEV unexpectedly already contains regime_id."
    )

FD004_DEV_CORRUPTION_BASE_630V17A = (
    raw_authority_630v17a
    .copy(deep=True)
    .reset_index(drop=True)
)

FD004_DEV_CORRUPTION_BASE_630V17A[
    "regime_id"
] = regime_ids_630v17a.copy()


# =============================================================================
# 4) EXACT CORRUPTION-BASE CHECKS
# =============================================================================

expected_raw_reset_630v17a = (
    raw_snapshot_630v17a
    .copy(deep=True)
    .reset_index(drop=True)
)

expected_columns_630v17a = tuple(
    list(
        expected_raw_reset_630v17a.columns
    )
    + ["regime_id"]
)

base_columns_exact_630v17a = bool(
    tuple(
        FD004_DEV_CORRUPTION_BASE_630V17A.columns
    )
    ==
    expected_columns_630v17a
)

base_raw_exact_630v17a = bool(
    FD004_DEV_CORRUPTION_BASE_630V17A[
        list(
            expected_raw_reset_630v17a.columns
        )
    ].equals(
        expected_raw_reset_630v17a
    )
)

base_regime_exact_630v17a = bool(
    np.array_equal(
        FD004_DEV_CORRUPTION_BASE_630V17A[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_snapshot_630v17a,
    )
)

base_rangeindex_exact_630v17a = bool(
    isinstance(
        FD004_DEV_CORRUPTION_BASE_630V17A.index,
        pd.RangeIndex,
    )
    and
    FD004_DEV_CORRUPTION_BASE_630V17A.index.start == 0
    and
    FD004_DEV_CORRUPTION_BASE_630V17A.index.stop
    == len(
        FD004_DEV_CORRUPTION_BASE_630V17A
    )
    and
    FD004_DEV_CORRUPTION_BASE_630V17A.index.step == 1
)

base_key_unique_630v17a = bool(
    not FD004_DEV_CORRUPTION_BASE_630V17A[
        ["unit_id", "cycle"]
    ].duplicated().any()
)

base_key_order_exact_630v17a = bool(
    np.array_equal(
        FD004_DEV_CORRUPTION_BASE_630V17A[
            ["unit_id", "cycle"]
        ].to_numpy(),
        expected_raw_reset_630v17a[
            ["unit_id", "cycle"]
        ].to_numpy(),
    )
)

raw_authority_unchanged_630v17a = bool(
    raw_authority_630v17a.equals(
        raw_snapshot_630v17a
    )
)

regime_authority_unchanged_630v17a = bool(
    np.array_equal(
        np.asarray(
            FD004_DEV_REGIME_IDS_630F,
            dtype=np.int64,
        ).reshape(-1),
        regime_snapshot_630v17a,
    )
)

if not all(
    (
        base_columns_exact_630v17a,
        base_raw_exact_630v17a,
        base_regime_exact_630v17a,
        base_rangeindex_exact_630v17a,
        base_key_unique_630v17a,
        base_key_order_exact_630v17a,
        raw_authority_unchanged_630v17a,
        regime_authority_unchanged_630v17a,
    )
):
    raise RuntimeError(
        "FD004 corruption-base construction failed exact checks."
    )


# =============================================================================
# 5) CORRECTED OPERATOR REGISTRY AUTHORITY
# =============================================================================

operator_registry_630v17a = (
    FD004_CORRUPTION_OPERATORS_630V13
)

expected_families_630v17a = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

registry_keys_exact_630v17a = bool(
    set(
        operator_registry_630v17a.keys()
    )
    ==
    set(
        expected_families_630v17a
    )
)

registry_all_callable_630v17a = bool(
    all(
        callable(
            operator_registry_630v17a[
                family_630v17a
            ]
        )
        for family_630v17a
        in expected_families_630v17a
    )
)

if not (
    registry_keys_exact_630v17a
    and registry_all_callable_630v17a
):
    raise RuntimeError(
        "Corrected FD004 operator registry authority changed."
    )


# =============================================================================
# 6) BIND EXACT HISTORICAL-ROUTING FD004 DISPATCHER
#
# Definition only.
# The function body is NOT called in this step.
# =============================================================================

def apply_manifest_scenario_fd004_630v17a(
    base_raw,
    scenario_row,
):
    family_630v17a = str(
        scenario_row[
            "family"
        ]
    )

    sensor_630v17a = str(
        scenario_row[
            "sensor"
        ]
    )

    contract_630v17a = (
        FD004_CORRUPTION_OPERATORS_630V13
    )

    if family_630v17a == "bias":

        corrupted_630v17a, _ = (
            contract_630v17a[
                "bias"
            ](
                base_raw,
                sensor_630v17a,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )

    elif family_630v17a == "noise":

        corrupted_630v17a, _, _ = (
            contract_630v17a[
                "noise"
            ](
                base_raw,
                sensor_630v17a,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )

    elif family_630v17a == "drift":

        corrupted_630v17a, _, _ = (
            contract_630v17a[
                "drift"
            ](
                base_raw,
                sensor_630v17a,
                float(
                    scenario_row[
                        "alpha"
                    ]
                ),
                int(
                    scenario_row[
                        "direction"
                    ]
                ),
            )
        )

    elif family_630v17a == "dropout":

        corrupted_630v17a, _, _ = (
            contract_630v17a[
                "dropout"
            ](
                base_raw,
                sensor_630v17a,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
                int(
                    scenario_row[
                        "seed"
                    ]
                ),
            )
        )

    elif family_630v17a == "stuck":

        corrupted_630v17a, _, _ = (
            contract_630v17a[
                "stuck"
            ](
                base_raw,
                sensor_630v17a,
                float(
                    scenario_row[
                        "requested_fraction"
                    ]
                ),
            )
        )

    else:
        raise ValueError(
            "Unknown corruption family: "
            + family_630v17a
        )

    return corrupted_630v17a


# =============================================================================
# 7) STATIC ROUTING CONTRACT — STILL NO DISPATCHER CALL
# =============================================================================

dispatcher_signature_630v17a = str(
    inspect.signature(
        apply_manifest_scenario_fd004_630v17a
    )
)

if dispatcher_signature_630v17a != (
    "(base_raw, scenario_row)"
):
    raise RuntimeError(
        "FD004 dispatcher signature differs from historical contract."
    )

FD004_DISPATCHER_ROUTING_SPEC_630V17A = (
    MappingProxyType(
        {
            "bias":
                (
                    ("alpha", "float"),
                    ("direction", "int"),
                ),

            "noise":
                (
                    ("alpha", "float"),
                    ("seed", "int"),
                ),

            "drift":
                (
                    ("alpha", "float"),
                    ("direction", "int"),
                ),

            "dropout":
                (
                    (
                        "requested_fraction",
                        "float",
                    ),
                    ("seed", "int"),
                ),

            "stuck":
                (
                    (
                        "requested_fraction",
                        "float",
                    ),
                ),
        }
    )
)


FD004_CORRUPTION_MODEL_BRIDGE_CONTRACT_630V17A = (
    MappingProxyType(
        {
            "apply_manifest_scenario":
                apply_manifest_scenario_fd004_630v17a,

            "operator_registry":
                operator_registry_630v17a,

            "corruption_base":
                FD004_DEV_CORRUPTION_BASE_630V17A,

            "routing_spec":
                FD004_DISPATCHER_ROUTING_SPEC_630V17A,

            "historical_dispatcher_function_sha256":
                str(
                    FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B
                ),

            "historical_fullrun_alias_audit_sha256":
                str(
                    FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1
                ),

            "positional_operator_contract_sha256":
                str(
                    FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
                ),

            "five_family_structural_validation_sha256":
                str(
                    FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15
                ),
        }
    )
)


FD004_APPLY_SCENARIO_630V17A = (
    FD004_CORRUPTION_MODEL_BRIDGE_CONTRACT_630V17A[
        "apply_manifest_scenario"
    ]
)


# =============================================================================
# 8) EVIDENCE FINGERPRINTS
# =============================================================================

def _sha_array_630v17a(
    value_630v17a
):
    arr_630v17a = np.ascontiguousarray(
        np.asarray(
            value_630v17a
        )
    )

    h_630v17a = hashlib.sha256()

    h_630v17a.update(
        str(
            arr_630v17a.dtype
        ).encode("utf-8")
    )

    h_630v17a.update(
        str(
            tuple(
                arr_630v17a.shape
            )
        ).encode("utf-8")
    )

    h_630v17a.update(
        arr_630v17a.tobytes(
            order="C"
        )
    )

    return h_630v17a.hexdigest()


FD004_CORRUPTION_BASE_REGIME_SHA256_630V17A = (
    _sha_array_630v17a(
        FD004_DEV_CORRUPTION_BASE_630V17A[
            "regime_id"
        ].to_numpy(
            dtype=np.int64
        )
    )
)

FD004_CORRUPTION_BASE_KEY_SHA256_630V17A = (
    _sha_array_630v17a(
        FD004_DEV_CORRUPTION_BASE_630V17A[
            ["unit_id", "cycle"]
        ].to_numpy(
            dtype=np.int64
        )
    )
)


binding_payload_630v17a = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-17A",

    "status":
        "CORRUPTION_BASE_AND_DISPATCHER_BOUND_NOT_EXECUTED",

    "corruption_base_rule":
        (
            "DEV raw deep copy -> reset_index(drop=True) -> "
            "append frozen aligned regime_id"
        ),

    "base_shape":
        tuple(
            FD004_DEV_CORRUPTION_BASE_630V17A.shape
        ),

    "base_columns":
        tuple(
            str(
                col_630v17a
            )
            for col_630v17a
            in FD004_DEV_CORRUPTION_BASE_630V17A.columns
        ),

    "base_columns_exact":
        base_columns_exact_630v17a,

    "base_raw_exact":
        base_raw_exact_630v17a,

    "base_regime_exact":
        base_regime_exact_630v17a,

    "base_rangeindex_exact":
        base_rangeindex_exact_630v17a,

    "base_key_unique":
        base_key_unique_630v17a,

    "base_key_order_exact":
        base_key_order_exact_630v17a,

    "raw_authority_unchanged":
        raw_authority_unchanged_630v17a,

    "regime_authority_unchanged":
        regime_authority_unchanged_630v17a,

    "regime_sha256":
        FD004_CORRUPTION_BASE_REGIME_SHA256_630V17A,

    "key_sha256":
        FD004_CORRUPTION_BASE_KEY_SHA256_630V17A,

    "dispatcher_signature":
        dispatcher_signature_630v17a,

    "routing_spec":
        {
            family_630v17a:
                tuple(
                    tuple(
                        item_630v17a
                    )
                    for item_630v17a
                    in FD004_DISPATCHER_ROUTING_SPEC_630V17A[
                        family_630v17a
                    ]
                )
            for family_630v17a
            in expected_families_630v17a
        },

    "return_semantics":
        "corrupted_dataframe_only",

    "historical_dispatcher_function_sha256":
        str(
            FD004_HISTORICAL_DISPATCHER_EXACT_SOURCE_SHA256_630V16B
        ),

    "historical_base_context_sha256":
        str(
            FD004_HISTORICAL_CORRUPTION_BASE_CONTEXT_SHA256_630V16D
        ),

    "historical_fullrun_alias_audit_sha256":
        str(
            FD004_HISTORICAL_FULLRUN_ALIAS_AUDIT_SHA256_630V16ER1
        ),

    "positional_operator_contract_sha256":
        str(
            FD004_POSITIONAL_OPERATOR_CONTRACT_SHA256_630V13
        ),

    "five_family_validation_sha256":
        str(
            FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15
        ),

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "preprocessing_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_BASE_DISPATCHER_BINDING_CONTRACT_630V17A = (
    MappingProxyType(
        binding_payload_630v17a
    )
)

FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A = (
    hashlib.sha256(
        json.dumps(
            binding_payload_630v17a,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 9) LOCK FLAGS
# =============================================================================

FD004_CORRUPTION_BASE_LOCKED_630V17A = True
FD004_DISPATCHER_BOUND_630V17A = True

FD004_DISPATCHER_EXECUTED_630V17A = False
FD004_CORRUPTION_EXECUTED_630V17A = False
FD004_FULL_CORRUPTION_EXECUTED_630V17A = False

FD004_PREPROCESSING_EXECUTED_630V17A = False
FD004_MODEL_INFERENCE_PERFORMED_630V17A = False
FD004_MC_EXECUTION_PERFORMED_630V17A = False

FD004_DEV_TARGETS_ERRORS_USED_630V17A = False
FD004_THRESHOLD_SELECTED_630V17A = False
FD004_POLICY_SELECTED_630V17A = False

FD004_TEST_CONTENTS_READ_630V17A = False
FD004_RUL_CONTENTS_READ_630V17A = False

all_checks_passed_630v17a = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-17A — FD004 CORRUPTION BASE + "
    "EXACT HISTORICAL-ROUTING DISPATCHER BIND"
)

print("=" * 176)

print("Dispatcher bound                  :", True)
print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Full 381-scenario run executed    :", False)
print("Preprocessing executed            :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FD004 CORRUPTION BASE ---")

print(
    "Shape                             :",
    FD004_DEV_CORRUPTION_BASE_630V17A.shape,
)

print(
    "Raw columns + regime_id exact     :",
    base_columns_exact_630v17a,
)

print(
    "Raw values exact                  :",
    base_raw_exact_630v17a,
)

print(
    "Frozen regime IDs exact           :",
    base_regime_exact_630v17a,
)

print(
    "RangeIndex exact                  :",
    base_rangeindex_exact_630v17a,
)

print(
    "unit_id/cycle unique              :",
    base_key_unique_630v17a,
)

print(
    "unit_id/cycle order exact         :",
    base_key_order_exact_630v17a,
)

print(
    "DEV raw authority unchanged       :",
    raw_authority_unchanged_630v17a,
)

print(
    "DEV regime authority unchanged    :",
    regime_authority_unchanged_630v17a,
)

print(
    "Regime vector SHA-256             :",
    FD004_CORRUPTION_BASE_REGIME_SHA256_630V17A,
)

print(
    "Row-key SHA-256                   :",
    FD004_CORRUPTION_BASE_KEY_SHA256_630V17A,
)

print("\n--- DISPATCHER ROUTING ---")

print(
    "Signature                         :",
    dispatcher_signature_630v17a,
)

for family_630v17a in expected_families_630v17a:
    print(
        f"{family_630v17a:8s}:",
        FD004_DISPATCHER_ROUTING_SPEC_630V17A[
            family_630v17a
        ],
    )

print(
    "Return semantics                  : corrupted DataFrame only"
)

print("\n--- EVIDENCE LOCK ---")

print(
    "Historical dispatcher SHA exact   :",
    True,
)

print(
    "Historical base context exact     :",
    True,
)

print(
    "Historical alias audit exact      :",
    True,
)

print(
    "Binding contract SHA-256          :",
    FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A,
)

print(
    "\nPASS — FD004 corruption base and dispatcher are now bound "
    "from the exact recovered historical authorities. "
    "No scenario was executed in this step."
)

print(
    "Next: validate dispatcher routing on the five already-locked "
    "structural scenarios before any 381-scenario execution."
)

In [ ]:
# STEP 6.30V-17B0 — FD004 RESIDENT FIVE-FAMILY REFERENCE AVAILABILITY AUDIT
#
# ZERO EXECUTION:
# - no dispatcher call
# - no corruption operator call
# - no preprocessing/model/MC/outcomes/TEST/RUL

import numpy as np
import pandas as pd
import hashlib
import json

required_630v17b0 = (
    "all_checks_passed_630v17a",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_RAW_630C",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A",
)

missing_630v17b0 = [
    name_630v17b0
    for name_630v17b0 in required_630v17b0
    if name_630v17b0 not in globals()
]

if missing_630v17b0:
    raise RuntimeError(
        "Missing STEP 6.30V-17B0 authority object(s): "
        + ", ".join(missing_630v17b0)
    )

if not bool(all_checks_passed_630v17a):
    raise RuntimeError(
        "STEP 6.30V-17A must remain passed."
    )

reference_names_630v17b0 = {
    "bias": "corrupted_630v14b1",
    "noise": "corrupted_630v14b2",
    "drift": "corrupted_630v14b3",
    "dropout": "corrupted_630v14b4",
    "stuck": "corrupted_630v14b5a",
}

reference_status_630v17b0 = {}

raw_cols_630v17b0 = tuple(
    str(c)
    for c in FD004_DEV_RAW_630C.columns
)

for family_630v17b0, name_630v17b0 in reference_names_630v17b0.items():

    present_630v17b0 = bool(
        name_630v17b0 in globals()
    )

    status_630v17b0 = {
        "object_name": name_630v17b0,
        "present": present_630v17b0,
    }

    if present_630v17b0:
        obj_630v17b0 = globals()[name_630v17b0]

        is_df_630v17b0 = isinstance(
            obj_630v17b0,
            pd.DataFrame,
        )

        status_630v17b0.update(
            {
                "is_dataframe": is_df_630v17b0,

                "shape": (
                    tuple(obj_630v17b0.shape)
                    if is_df_630v17b0
                    else None
                ),

                "index_type": (
                    type(obj_630v17b0.index).__name__
                    if is_df_630v17b0
                    else None
                ),

                "columns": (
                    tuple(
                        str(c)
                        for c in obj_630v17b0.columns
                    )
                    if is_df_630v17b0
                    else None
                ),

                "has_all_raw_columns": (
                    bool(
                        set(raw_cols_630v17b0).issubset(
                            set(
                                str(c)
                                for c in obj_630v17b0.columns
                            )
                        )
                    )
                    if is_df_630v17b0
                    else False
                ),

                "has_regime_id": (
                    bool(
                        "regime_id"
                        in obj_630v17b0.columns
                    )
                    if is_df_630v17b0
                    else False
                ),

                "row_count_matches": (
                    bool(
                        len(obj_630v17b0)
                        == len(FD004_DEV_RAW_630C)
                    )
                    if is_df_630v17b0
                    else False
                ),
            }
        )

    reference_status_630v17b0[
        family_630v17b0
    ] = status_630v17b0


all_present_630v17b0 = bool(
    all(
        item_630v17b0["present"]
        for item_630v17b0
        in reference_status_630v17b0.values()
    )
)

all_dataframes_630v17b0 = bool(
    all(
        item_630v17b0.get(
            "is_dataframe",
            False,
        )
        for item_630v17b0
        in reference_status_630v17b0.values()
    )
)

all_raw_columns_630v17b0 = bool(
    all(
        item_630v17b0.get(
            "has_all_raw_columns",
            False,
        )
        for item_630v17b0
        in reference_status_630v17b0.values()
    )
)

all_row_counts_630v17b0 = bool(
    all(
        item_630v17b0.get(
            "row_count_matches",
            False,
        )
        for item_630v17b0
        in reference_status_630v17b0.values()
    )
)


scenario_rows_status_630v17b0 = {}

for family_630v17b0 in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    row_630v17b0 = (
        FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
            family_630v17b0
        ]
    )

    scenario_rows_status_630v17b0[
        family_630v17b0
    ] = {
        "scenario_id":
            str(
                row_630v17b0[
                    "scenario_id"
                ]
            ),

        "scenario_number":
            int(
                row_630v17b0[
                    "scenario_number"
                ]
            ),

        "family":
            str(
                row_630v17b0[
                    "family"
                ]
            ),

        "sensor":
            str(
                row_630v17b0[
                    "sensor"
                ]
            ),
    }


base_snapshot_630v17b0 = (
    FD004_DEV_CORRUPTION_BASE_630V17A
    .copy(deep=True)
)

base_unchanged_630v17b0 = bool(
    FD004_DEV_CORRUPTION_BASE_630V17A.equals(
        base_snapshot_630v17b0
    )
)


FD004_RESIDENT_REFERENCE_STATUS_630V17B0 = (
    reference_status_630v17b0
)

FD004_RESIDENT_REFERENCE_OBJECT_NAMES_630V17B0 = (
    dict(
        reference_names_630v17b0
    )
)


FD004_RESIDENT_REFERENCE_AUDIT_630V17B0 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-17B0",

    "reference_status":
        reference_status_630v17b0,

    "scenario_rows":
        scenario_rows_status_630v17b0,

    "all_five_reference_objects_present":
        all_present_630v17b0,

    "all_five_are_dataframes":
        all_dataframes_630v17b0,

    "all_five_have_raw_columns":
        all_raw_columns_630v17b0,

    "all_five_row_counts_match":
        all_row_counts_630v17b0,

    "dispatcher_base_unchanged":
        base_unchanged_630v17b0,

    "dispatcher_executed":
        False,

    "corruption_executed":
        False,

    "preprocessing_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_RESIDENT_REFERENCE_AUDIT_SHA256_630V17B0 = (
    hashlib.sha256(
        json.dumps(
            FD004_RESIDENT_REFERENCE_AUDIT_630V17B0,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DISPATCHER_EXECUTED_630V17B0 = False
FD004_CORRUPTION_EXECUTED_630V17B0 = False
FD004_MODEL_INFERENCE_PERFORMED_630V17B0 = False
FD004_MC_EXECUTION_PERFORMED_630V17B0 = False
FD004_DEV_TARGETS_ERRORS_USED_630V17B0 = False
FD004_THRESHOLD_SELECTED_630V17B0 = False
FD004_POLICY_SELECTED_630V17B0 = False
FD004_TEST_CONTENTS_READ_630V17B0 = False
FD004_RUL_CONTENTS_READ_630V17B0 = False


all_checks_passed_630v17b0 = bool(
    all_present_630v17b0
    and all_dataframes_630v17b0
    and all_raw_columns_630v17b0
    and all_row_counts_630v17b0
    and base_unchanged_630v17b0
)


print("=" * 176)

print(
    "STEP 6.30V-17B0 — FD004 RESIDENT FIVE-FAMILY "
    "REFERENCE AVAILABILITY AUDIT"
)

print("=" * 176)

print("Dispatcher executed               :", False)
print("Corruption executed               :", False)
print("Preprocessing executed            :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- RESIDENT DIRECT-OPERATOR REFERENCES ---")

for family_630v17b0 in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    item_630v17b0 = (
        reference_status_630v17b0[
            family_630v17b0
        ]
    )

    print(
        f"{family_630v17b0:8s}: "
        f"name={item_630v17b0['object_name']!r}, "
        f"present={item_630v17b0['present']}, "
        f"shape={item_630v17b0.get('shape')}, "
        f"index={item_630v17b0.get('index_type')}, "
        f"regime_id={item_630v17b0.get('has_regime_id')}"
    )

print("\n--- COMPARABILITY CHECKS ---")

print(
    "All five references present       :",
    all_present_630v17b0,
)

print(
    "All five are DataFrames           :",
    all_dataframes_630v17b0,
)

print(
    "All contain raw columns           :",
    all_raw_columns_630v17b0,
)

print(
    "All row counts match DEV          :",
    all_row_counts_630v17b0,
)

print(
    "Dispatcher base unchanged         :",
    base_unchanged_630v17b0,
)

print("\n--- LOCKED SCENARIOS ---")

for (
    family_630v17b0,
    item_630v17b0,
) in scenario_rows_status_630v17b0.items():

    print(
        f"{family_630v17b0:8s}: "
        f"{item_630v17b0['scenario_id']} / "
        f"{item_630v17b0['scenario_number']} / "
        f"{item_630v17b0['sensor']}"
    )

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Resident-reference audit SHA-256  :",
    FD004_RESIDENT_REFERENCE_AUDIT_SHA256_630V17B0,
)

if all_checks_passed_630v17b0:

    print(
        "\nPASS — all five already-proven direct-operator outputs "
        "remain resident and are suitable as routing references. "
        "Nothing was executed."
    )

    print(
        "Next: execute the bound dispatcher once for each of those "
        "five locked scenarios and compare rowwise against these "
        "resident references."
    )

else:

    print(
        "\nSTOP — one or more resident routing-reference objects are "
        "unavailable or structurally unsuitable. Nothing was executed."
    )

    print(
        "Next: adapt the routing validation from the exact availability "
        "report above without rerunning anything unnecessarily."
    )

In [ ]:
# STEP 6.30V-17B1 — FD004 FIVE-FAMILY DISPATCHER ROUTING VALIDATION
#
# EXACTLY FIVE dispatcher executions:
# one locked scenario per corruption family.
#
# NO full 381-scenario run.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v17b1 = (
    "all_checks_passed_630v17b0",
    "FD004_APPLY_SCENARIO_630V17A",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A",
    "FD004_RESIDENT_REFERENCE_OBJECT_NAMES_630V17B0",
    "FD004_RESIDENT_REFERENCE_AUDIT_SHA256_630V17B0",
    "FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A",
)

missing_630v17b1 = [
    name_630v17b1
    for name_630v17b1 in required_630v17b1
    if name_630v17b1 not in globals()
]

if missing_630v17b1:
    raise RuntimeError(
        "Missing STEP 6.30V-17B1 authority object(s): "
        + ", ".join(missing_630v17b1)
    )

if not bool(all_checks_passed_630v17b0):
    raise RuntimeError(
        "STEP 6.30V-17B0 must remain passed."
    )

if not callable(FD004_APPLY_SCENARIO_630V17A):
    raise RuntimeError(
        "Bound FD004 dispatcher is not callable."
    )


# =============================================================================
# 1) IMMUTABLE VALIDATION AUTHORITIES
# =============================================================================

family_order_630v17b1 = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

base_630v17b1 = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v17b1 = (
    base_630v17b1.copy(deep=True)
)

base_columns_630v17b1 = tuple(
    base_630v17b1.columns
)

raw_columns_630v17b1 = tuple(
    col_630v17b1
    for col_630v17b1
    in base_columns_630v17b1
    if col_630v17b1 != "regime_id"
)

base_regime_630v17b1 = (
    base_630v17b1[
        "regime_id"
    ].to_numpy(dtype=np.int64)
)


# =============================================================================
# 2) SHA HELPER
# =============================================================================

def _sha_numeric_frame_630v17b1(
    frame_630v17b1,
    columns_630v17b1,
):
    arr_630v17b1 = np.ascontiguousarray(
        frame_630v17b1[
            list(columns_630v17b1)
        ].to_numpy(dtype=np.float64)
    )

    h_630v17b1 = hashlib.sha256()

    h_630v17b1.update(
        str(arr_630v17b1.dtype).encode("utf-8")
    )

    h_630v17b1.update(
        str(
            tuple(
                arr_630v17b1.shape
            )
        ).encode("utf-8")
    )

    h_630v17b1.update(
        arr_630v17b1.tobytes(order="C")
    )

    return h_630v17b1.hexdigest()


# =============================================================================
# 3) EXECUTE BOUND DISPATCHER EXACTLY ONCE PER LOCKED FAMILY SCENARIO
# =============================================================================

dispatcher_results_630v17b1 = {}
routing_checks_630v17b1 = {}

dispatcher_execution_count_630v17b1 = 0

for family_630v17b1 in family_order_630v17b1:

    scenario_row_630v17b1 = (
        FD004_STRUCTURAL_VALIDATION_SCENARIO_ROWS_630V14A[
            family_630v17b1
        ]
    )

    if str(
        scenario_row_630v17b1[
            "family"
        ]
    ) != family_630v17b1:
        raise RuntimeError(
            "Locked structural-validation scenario family changed: "
            + family_630v17b1
        )

    reference_name_630v17b1 = (
        FD004_RESIDENT_REFERENCE_OBJECT_NAMES_630V17B0[
            family_630v17b1
        ]
    )

    if reference_name_630v17b1 not in globals():
        raise RuntimeError(
            "Resident direct-operator reference disappeared: "
            + reference_name_630v17b1
        )

    reference_630v17b1 = globals()[
        reference_name_630v17b1
    ]

    if not isinstance(
        reference_630v17b1,
        pd.DataFrame,
    ):
        raise RuntimeError(
            "Resident direct-operator reference is no longer a DataFrame: "
            + reference_name_630v17b1
        )

    # EXACTLY ONE dispatcher call for this family.
    dispatched_630v17b1 = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v17b1,
            scenario_row_630v17b1,
        )
    )

    dispatcher_execution_count_630v17b1 += 1

    if not isinstance(
        dispatched_630v17b1,
        pd.DataFrame,
    ):
        raise RuntimeError(
            "Dispatcher did not return a DataFrame for family: "
            + family_630v17b1
        )

    shape_exact_630v17b1 = bool(
        dispatched_630v17b1.shape
        ==
        base_630v17b1.shape
    )

    columns_exact_630v17b1 = bool(
        tuple(
            dispatched_630v17b1.columns
        )
        ==
        base_columns_630v17b1
    )

    rangeindex_exact_630v17b1 = bool(
        isinstance(
            dispatched_630v17b1.index,
            pd.RangeIndex,
        )
        and
        dispatched_630v17b1.index.start == 0
        and
        dispatched_630v17b1.index.stop
        == len(dispatched_630v17b1)
        and
        dispatched_630v17b1.index.step == 1
    )

    key_order_exact_630v17b1 = bool(
        np.array_equal(
            dispatched_630v17b1[
                ["unit_id", "cycle"]
            ].to_numpy(),
            base_630v17b1[
                ["unit_id", "cycle"]
            ].to_numpy(),
        )
    )

    regime_exact_630v17b1 = bool(
        np.array_equal(
            dispatched_630v17b1[
                "regime_id"
            ].to_numpy(dtype=np.int64),
            base_regime_630v17b1,
        )
    )

    reference_has_raw_columns_630v17b1 = bool(
        set(
            raw_columns_630v17b1
        ).issubset(
            set(
                reference_630v17b1.columns
            )
        )
    )

    if not reference_has_raw_columns_630v17b1:
        raise RuntimeError(
            "Reference is missing one or more raw columns for family: "
            + family_630v17b1
        )

    # Compare raw-column VALUES only because the historical execution
    # base intentionally uses RangeIndex, whereas prior direct-operator
    # references preserve the original caller index.
    raw_values_exact_to_reference_630v17b1 = bool(
        np.array_equal(
            dispatched_630v17b1[
                list(
                    raw_columns_630v17b1
                )
            ].to_numpy(),
            reference_630v17b1[
                list(
                    raw_columns_630v17b1
                )
            ].to_numpy(),
            equal_nan=True,
        )
    )

    # Dropout's earlier direct reference already contains regime_id.
    if "regime_id" in reference_630v17b1.columns:
        reference_regime_exact_630v17b1 = bool(
            np.array_equal(
                dispatched_630v17b1[
                    "regime_id"
                ].to_numpy(dtype=np.int64),
                reference_630v17b1[
                    "regime_id"
                ].to_numpy(dtype=np.int64),
            )
        )
    else:
        reference_regime_exact_630v17b1 = True

    output_all_finite_630v17b1 = bool(
        np.all(
            np.isfinite(
                dispatched_630v17b1[
                    list(
                        base_columns_630v17b1
                    )
                ].to_numpy(dtype=np.float64)
            )
        )
    )

    scenario_pass_630v17b1 = bool(
        shape_exact_630v17b1
        and columns_exact_630v17b1
        and rangeindex_exact_630v17b1
        and key_order_exact_630v17b1
        and regime_exact_630v17b1
        and raw_values_exact_to_reference_630v17b1
        and reference_regime_exact_630v17b1
        and output_all_finite_630v17b1
    )

    if not scenario_pass_630v17b1:
        raise RuntimeError(
            "Dispatcher routing validation failed for family: "
            + family_630v17b1
        )

    output_sha_630v17b1 = (
        _sha_numeric_frame_630v17b1(
            dispatched_630v17b1,
            base_columns_630v17b1,
        )
    )

    dispatcher_results_630v17b1[
        family_630v17b1
    ] = dispatched_630v17b1

    routing_checks_630v17b1[
        family_630v17b1
    ] = {
        "scenario_id":
            str(
                scenario_row_630v17b1[
                    "scenario_id"
                ]
            ),

        "scenario_number":
            int(
                scenario_row_630v17b1[
                    "scenario_number"
                ]
            ),

        "sensor":
            str(
                scenario_row_630v17b1[
                    "sensor"
                ]
            ),

        "reference_object":
            reference_name_630v17b1,

        "shape_exact":
            shape_exact_630v17b1,

        "columns_exact":
            columns_exact_630v17b1,

        "rangeindex_exact":
            rangeindex_exact_630v17b1,

        "unit_cycle_order_exact":
            key_order_exact_630v17b1,

        "regime_id_exact":
            regime_exact_630v17b1,

        "raw_values_exact_to_direct_reference":
            raw_values_exact_to_reference_630v17b1,

        "reference_regime_exact_if_present":
            reference_regime_exact_630v17b1,

        "all_output_values_finite":
            output_all_finite_630v17b1,

        "output_sha256":
            output_sha_630v17b1,

        "passed":
            scenario_pass_630v17b1,
    }


# =============================================================================
# 4) SOURCE-MUTATION AUDIT
# =============================================================================

base_unchanged_after_five_630v17b1 = bool(
    base_630v17b1.equals(
        base_snapshot_630v17b1
    )
)

if not base_unchanged_after_five_630v17b1:
    raise RuntimeError(
        "Bound dispatcher mutated the frozen FD004 corruption base."
    )

if dispatcher_execution_count_630v17b1 != 5:
    raise RuntimeError(
        "Unexpected dispatcher execution count."
    )

all_five_routing_exact_630v17b1 = bool(
    all(
        routing_checks_630v17b1[
            family_630v17b1
        ][
            "passed"
        ]
        for family_630v17b1
        in family_order_630v17b1
    )
)

if not all_five_routing_exact_630v17b1:
    raise RuntimeError(
        "One or more five-family dispatcher-routing checks failed."
    )


# =============================================================================
# 5) RETAIN VALIDATION EVIDENCE
# =============================================================================

FD004_DISPATCHER_VALIDATION_OUTPUTS_630V17B1 = (
    dispatcher_results_630v17b1
)

FD004_DISPATCHER_ROUTING_CHECKS_630V17B1 = (
    routing_checks_630v17b1
)

FD004_DISPATCHER_ROUTING_VALIDATION_CONTRACT_630V17B1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-17B1",

    "status":
        "FIVE_FAMILY_DISPATCHER_ROUTING_VALIDATED",

    "dispatcher_execution_count":
        int(
            dispatcher_execution_count_630v17b1
        ),

    "validation_families":
        family_order_630v17b1,

    "routing_checks":
        routing_checks_630v17b1,

    "base_unchanged_after_five_dispatches":
        base_unchanged_after_five_630v17b1,

    "all_five_routing_exact":
        all_five_routing_exact_630v17b1,

    "resident_reference_audit_sha256":
        str(
            FD004_RESIDENT_REFERENCE_AUDIT_SHA256_630V17B0
        ),

    "dispatcher_binding_sha256":
        str(
            FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A
        ),

    "full_381_scenario_execution_started":
        False,

    "preprocessing_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_DISPATCHER_ROUTING_VALIDATION_SHA256_630V17B1 = (
    hashlib.sha256(
        json.dumps(
            FD004_DISPATCHER_ROUTING_VALIDATION_CONTRACT_630V17B1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 6) LOCK FLAGS
# =============================================================================

FD004_DISPATCHER_ROUTING_VALIDATED_630V17B1 = True

FD004_DISPATCHER_EXECUTION_COUNT_630V17B1 = (
    dispatcher_execution_count_630v17b1
)

FD004_CORRUPTION_EXECUTION_COUNT_630V17B1 = (
    dispatcher_execution_count_630v17b1
)

FD004_FULL_CORRUPTION_EXECUTED_630V17B1 = False

FD004_PREPROCESSING_EXECUTED_630V17B1 = False
FD004_MODEL_INFERENCE_PERFORMED_630V17B1 = False
FD004_MC_EXECUTION_PERFORMED_630V17B1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V17B1 = False
FD004_THRESHOLD_SELECTED_630V17B1 = False
FD004_POLICY_SELECTED_630V17B1 = False
FD004_TEST_CONTENTS_READ_630V17B1 = False
FD004_RUL_CONTENTS_READ_630V17B1 = False

all_checks_passed_630v17b1 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-17B1 — FD004 FIVE-FAMILY "
    "DISPATCHER ROUTING VALIDATION"
)

print("=" * 176)

print(
    "Dispatcher executions             :",
    dispatcher_execution_count_630v17b1,
)

print(
    "Full 381-scenario run executed    :",
    False,
)

print("Preprocessing executed            :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- ROUTING VALIDATION ---")

for family_630v17b1 in family_order_630v17b1:

    check_630v17b1 = (
        routing_checks_630v17b1[
            family_630v17b1
        ]
    )

    print(
        f"\n{family_630v17b1.upper()} — "
        f"{check_630v17b1['scenario_id']}"
    )

    print(
        "  Raw values exact to reference   :",
        check_630v17b1[
            "raw_values_exact_to_direct_reference"
        ],
    )

    print(
        "  regime_id exact                 :",
        check_630v17b1[
            "regime_id_exact"
        ],
    )

    print(
        "  unit_id/cycle order exact       :",
        check_630v17b1[
            "unit_cycle_order_exact"
        ],
    )

    print(
        "  Shape/columns exact             :",
        (
            check_630v17b1[
                "shape_exact"
            ],
            check_630v17b1[
                "columns_exact"
            ],
        ),
    )

    print(
        "  RangeIndex exact                :",
        check_630v17b1[
            "rangeindex_exact"
        ],
    )

    print(
        "  All output values finite        :",
        check_630v17b1[
            "all_output_values_finite"
        ],
    )

    print(
        "  Output SHA-256                  :",
        check_630v17b1[
            "output_sha256"
        ],
    )

print("\n--- GLOBAL CHECKS ---")

print(
    "All five routing checks exact     :",
    all_five_routing_exact_630v17b1,
)

print(
    "Frozen corruption base unchanged  :",
    base_unchanged_after_five_630v17b1,
)

print(
    "Routing validation SHA-256        :",
    FD004_DISPATCHER_ROUTING_VALIDATION_SHA256_630V17B1,
)

print(
    "\nPASS — the bound FD004 dispatcher reproduces the already-"
    "validated direct-operator corruption values for all five "
    "families while preserving regime IDs and the frozen corruption base."
)

print(
    "Next: perform a zero-execution admission audit for the frozen "
    "381-scenario execution contract before starting any family block."
)

In [ ]:
# STEP 6.30V-18A — FD004 ZERO-EXECUTION ADMISSION AUDIT
# for frozen 381-scenario controlled-corruption execution
#
# NO scenario execution.
# NO corruption.
# NO preprocessing.
# NO model inference.
# NO MC.
# NO DEV outcomes/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import pandas as pd
import torch
import inspect
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18a = (
    "all_checks_passed_630v17b1",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U",
    "FD004_CORRUPTION_MANIFEST_ANNOTATED_630U",
    "FD004_CORRUPTION_MANIFEST_SHA256_630S",
    "FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U",
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "MODEL_FD004_630I",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A",
    "FD004_DISPATCHER_ROUTING_VALIDATION_SHA256_630V17B1",
    "FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15",
)

missing_630v18a = [
    name
    for name in required_630v18a
    if name not in globals()
]

if missing_630v18a:
    raise RuntimeError(
        "Missing STEP 6.30V-18A authority object(s): "
        + ", ".join(missing_630v18a)
    )

if not bool(all_checks_passed_630v17b1):
    raise RuntimeError(
        "STEP 6.30V-17B1 must remain passed."
    )


# =============================================================================
# 1) FROZEN EVIDENCE HASH GUARDS
# =============================================================================

expected_hashes_630v18a = {
    "scientific_manifest":
        "a772f42919e117d2c889b395fcd27d8029a68505eec9238ae0f8b827c3758a08",

    "annotated_manifest":
        "202caf0404c4d1b7497120e57fa2c63a87a3a5e92fa567009ac938b078d2789d",

    "execution_manifest":
        "24f88b224a65538441d5e4fe3ced132b56dcbc03aa781d094856e952b2b58bba",

    "inadmissible_manifest":
        "68ebf365123c35d097c0940458eb5fe94288da4a3da17f1abd08aba75028532e",

    "full_execution_contract":
        "687f663209e062f0da2827911ddc38ee185c4d0e181e0c1dee6ccc7c5760e21a",

    "dispatcher_binding":
        "5fcbc508f5aec2207af4f6f83508fd2efbe8a34fa8eea085466e452473873294",

    "dispatcher_routing_validation":
        "14384507670ea4c58b3a8079d0ab0f85700a1fb94f6f01dd0e3c4e60fe194097",

    "five_family_closure":
        "4502914acc327be3c8c00cbfdaf5ea84319887a57d4e848b4fc53fe10cce3477",
}

observed_hashes_630v18a = {
    "scientific_manifest":
        str(
            FD004_CORRUPTION_MANIFEST_SHA256_630S
        ),

    "annotated_manifest":
        str(
            FD004_CORRUPTION_MANIFEST_ANNOTATED_SHA256_630U
        ),

    "execution_manifest":
        str(
            FD004_CORRUPTION_EXECUTION_MANIFEST_SHA256_630U
        ),

    "inadmissible_manifest":
        str(
            FD004_CORRUPTION_INADMISSIBLE_MANIFEST_SHA256_630U
        ),

    "full_execution_contract":
        str(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6
        ),

    "dispatcher_binding":
        str(
            FD004_CORRUPTION_BASE_DISPATCHER_BINDING_SHA256_630V17A
        ),

    "dispatcher_routing_validation":
        str(
            FD004_DISPATCHER_ROUTING_VALIDATION_SHA256_630V17B1
        ),

    "five_family_closure":
        str(
            FD004_FIVE_FAMILY_STRUCTURAL_VALIDATION_SHA256_630V15
        ),
}

hash_checks_630v18a = {
    key: bool(
        observed_hashes_630v18a[key]
        ==
        expected_hashes_630v18a[key]
    )
    for key in expected_hashes_630v18a
}

if not all(
    hash_checks_630v18a.values()
):
    failed_630v18a = [
        key
        for key, passed
        in hash_checks_630v18a.items()
        if not passed
    ]

    raise RuntimeError(
        "Frozen evidence hash mismatch: "
        + ", ".join(
            failed_630v18a
        )
    )


# =============================================================================
# 2) MANIFEST STRUCTURE
# =============================================================================

exec_manifest_630v18a = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

inadmissible_630v18a = (
    FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U
)

annotated_630v18a = (
    FD004_CORRUPTION_MANIFEST_ANNOTATED_630U
)

for name_630v18a, obj_630v18a in (
    (
        "execution",
        exec_manifest_630v18a,
    ),
    (
        "inadmissible",
        inadmissible_630v18a,
    ),
    (
        "annotated",
        annotated_630v18a,
    ),
):
    if not isinstance(
        obj_630v18a,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{name_630v18a} manifest is not a pandas DataFrame."
        )


required_cols_630v18a = {
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "alpha",
    "direction",
    "seed",
    "requested_fraction",
}

missing_cols_630v18a = sorted(
    required_cols_630v18a
    -
    set(
        exec_manifest_630v18a.columns
    )
)

if missing_cols_630v18a:
    raise RuntimeError(
        "Execution manifest missing routing column(s): "
        + ", ".join(
            missing_cols_630v18a
        )
    )


count_checks_630v18a = {
    "annotated_441":
        len(
            annotated_630v18a
        )
        == 441,

    "execution_381":
        len(
            exec_manifest_630v18a
        )
        == 381,

    "inadmissible_60":
        len(
            inadmissible_630v18a
        )
        == 60,

    "partition_total_441":
        (
            len(
                exec_manifest_630v18a
            )
            +
            len(
                inadmissible_630v18a
            )
        )
        == 441,
}

if not all(
    count_checks_630v18a.values()
):
    raise RuntimeError(
        "Manifest population counts changed."
    )


scenario_numbers_630v18a = (
    exec_manifest_630v18a[
        "scenario_number"
    ].to_numpy(
        dtype=np.int64
    )
)

scenario_ids_630v18a = (
    exec_manifest_630v18a[
        "scenario_id"
    ]
    .astype(str)
    .to_numpy()
)

families_630v18a = (
    exec_manifest_630v18a[
        "family"
    ]
    .astype(str)
    .to_numpy()
)


identity_checks_630v18a = {
    "execution_numbers_unique":
        (
            len(
                np.unique(
                    scenario_numbers_630v18a
                )
            )
            ==
            len(
                scenario_numbers_630v18a
            )
        ),

    "execution_ids_unique":
        (
            len(
                np.unique(
                    scenario_ids_630v18a
                )
            )
            ==
            len(
                scenario_ids_630v18a
            )
        ),

    "execution_numbers_strictly_increasing":
        bool(
            np.all(
                np.diff(
                    scenario_numbers_630v18a
                )
                > 0
            )
        ),

    "scenario_id_number_mapping_exact":
        bool(
            np.all(
                scenario_ids_630v18a
                ==
                np.asarray(
                    [
                        f"FD004_S{int(n):03d}"
                        for n
                        in scenario_numbers_630v18a
                    ],
                    dtype=object,
                )
            )
        ),
}


inad_numbers_630v18a = set(
    inadmissible_630v18a[
        "scenario_number"
    ]
    .to_numpy(
        dtype=np.int64
    )
    .tolist()
)

exec_numbers_set_630v18a = set(
    scenario_numbers_630v18a.tolist()
)

annot_numbers_630v18a = set(
    annotated_630v18a[
        "scenario_number"
    ]
    .to_numpy(
        dtype=np.int64
    )
    .tolist()
)


identity_checks_630v18a.update(
    {
        "exec_inadmissible_disjoint":
            exec_numbers_set_630v18a.isdisjoint(
                inad_numbers_630v18a
            ),

        "annotated_numbers_exact_1_to_441":
            (
                annot_numbers_630v18a
                ==
                set(
                    range(
                        1,
                        442,
                    )
                )
            ),

        "partition_matches_annotated":
            (
                exec_numbers_set_630v18a
                |
                inad_numbers_630v18a
            )
            ==
            annot_numbers_630v18a,
    }
)


if not all(
    identity_checks_630v18a.values()
):
    failed_630v18a = [
        key
        for key, passed
        in identity_checks_630v18a.items()
        if not passed
    ]

    raise RuntimeError(
        "Manifest identity check failed: "
        + ", ".join(
            failed_630v18a
        )
    )


# =============================================================================
# 3) FROZEN FAMILY BLOCKS
# =============================================================================

expected_block_spec_630v18a = {
    "bias": {
        "count": 102,
        "min": 7,
        "max": 126,
    },

    "noise": {
        "count": 51,
        "min": 130,
        "max": 189,
    },

    "drift": {
        "count": 102,
        "min": 196,
        "max": 315,
    },

    "dropout": {
        "count": 63,
        "min": 316,
        "max": 378,
    },

    "stuck": {
        "count": 63,
        "min": 379,
        "max": 441,
    },
}

family_order_630v18a = (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
)

family_block_checks_630v18a = {}

expected_family_sequence_630v18a = []

for family_630v18a in family_order_630v18a:

    spec_630v18a = (
        expected_block_spec_630v18a[
            family_630v18a
        ]
    )

    expected_family_sequence_630v18a.extend(
        [
            family_630v18a
        ]
        *
        spec_630v18a[
            "count"
        ]
    )

    rows_630v18a = (
        exec_manifest_630v18a.loc[
            exec_manifest_630v18a[
                "family"
            ].astype(str)
            ==
            family_630v18a
        ]
    )

    nums_630v18a = (
        rows_630v18a[
            "scenario_number"
        ].to_numpy(
            dtype=np.int64
        )
    )

    family_block_checks_630v18a[
        family_630v18a
    ] = {
        "count_exact":
            len(
                rows_630v18a
            )
            ==
            spec_630v18a[
                "count"
            ],

        "min_exact":
            (
                int(
                    nums_630v18a.min()
                )
                ==
                spec_630v18a[
                    "min"
                ]
                if len(
                    nums_630v18a
                )
                else False
            ),

        "max_exact":
            (
                int(
                    nums_630v18a.max()
                )
                ==
                spec_630v18a[
                    "max"
                ]
                if len(
                    nums_630v18a
                )
                else False
            ),
    }


family_sequence_exact_630v18a = bool(
    families_630v18a.tolist()
    ==
    expected_family_sequence_630v18a
)

if not family_sequence_exact_630v18a:
    raise RuntimeError(
        "Executable manifest family-block order changed."
    )


if not all(
    all(
        checks_630v18a.values()
    )
    for checks_630v18a
    in family_block_checks_630v18a.values()
):
    raise RuntimeError(
        "Executable manifest family-block count/range changed."
    )


inad_family_counts_630v18a = (
    inadmissible_630v18a[
        "family"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)

inad_family_exact_630v18a = bool(
    inad_family_counts_630v18a.get(
        "bias",
        0,
    )
    == 24
    and
    inad_family_counts_630v18a.get(
        "noise",
        0,
    )
    == 12
    and
    inad_family_counts_630v18a.get(
        "drift",
        0,
    )
    == 24
    and
    inad_family_counts_630v18a.get(
        "dropout",
        0,
    )
    == 0
    and
    inad_family_counts_630v18a.get(
        "stuck",
        0,
    )
    == 0
)

if not inad_family_exact_630v18a:
    raise RuntimeError(
        "Structurally inadmissible family counts changed."
    )


# =============================================================================
# 4) ROUTING FIELD COMPLETENESS
# =============================================================================

routing_field_checks_630v18a = {}

for family_630v18a in family_order_630v18a:

    rows_630v18a = (
        exec_manifest_630v18a.loc[
            exec_manifest_630v18a[
                "family"
            ].astype(str)
            ==
            family_630v18a
        ]
    )

    base_ok_630v18a = bool(
        rows_630v18a[
            "scenario_id"
        ].notna().all()
        and
        rows_630v18a[
            "scenario_number"
        ].notna().all()
        and
        rows_630v18a[
            "sensor"
        ].notna().all()
    )

    if family_630v18a in (
        "bias",
        "drift",
    ):

        family_ok_630v18a = bool(
            rows_630v18a[
                "alpha"
            ].notna().all()
            and
            rows_630v18a[
                "direction"
            ].notna().all()
            and
            rows_630v18a[
                "direction"
            ]
            .astype(float)
            .isin(
                [
                    -1.0,
                    1.0,
                ]
            )
            .all()
        )

    elif family_630v18a == "noise":

        family_ok_630v18a = bool(
            rows_630v18a[
                "alpha"
            ].notna().all()
            and
            rows_630v18a[
                "seed"
            ].notna().all()
        )

    elif family_630v18a == "dropout":

        family_ok_630v18a = bool(
            rows_630v18a[
                "requested_fraction"
            ].notna().all()
            and
            rows_630v18a[
                "seed"
            ].notna().all()
        )

    elif family_630v18a == "stuck":

        family_ok_630v18a = bool(
            rows_630v18a[
                "requested_fraction"
            ].notna().all()
        )

    else:
        family_ok_630v18a = False

    routing_field_checks_630v18a[
        family_630v18a
    ] = bool(
        base_ok_630v18a
        and
        family_ok_630v18a
    )


if not all(
    routing_field_checks_630v18a.values()
):
    failed_630v18a = [
        key
        for key, passed
        in routing_field_checks_630v18a.items()
        if not passed
    ]

    raise RuntimeError(
        "Manifest routing fields incomplete: "
        + ", ".join(
            failed_630v18a
        )
    )


# =============================================================================
# 5) LIVE INTERFACE AUDIT — NOTHING CALLED
# =============================================================================

callable_authorities_630v18a = {
    "dispatcher":
        FD004_APPLY_SCENARIO_630V17A,

    "normalize":
        normalize_fd004_raw_630v2,

    "build_windows":
        build_windows_fd004_630v2,

    "mc":
        fd004_mc_dropout_predict_630v4r2,

    "reliability":
        apply_fd004_final_reliability_chain_630v4r2,
}


for name_630v18a, fn_630v18a in (
    callable_authorities_630v18a.items()
):
    if not callable(
        fn_630v18a
    ):
        raise RuntimeError(
            f"{name_630v18a} authority is not callable."
        )


signatures_630v18a = {
    name_630v18a:
        str(
            inspect.signature(
                fn_630v18a
            )
        )
    for name_630v18a, fn_630v18a
    in callable_authorities_630v18a.items()
}

parameter_counts_630v18a = {
    name_630v18a:
        len(
            inspect.signature(
                fn_630v18a
            ).parameters
        )
    for name_630v18a, fn_630v18a
    in callable_authorities_630v18a.items()
}

expected_parameter_counts_630v18a = {
    "dispatcher": 2,
    "normalize": 2,
    "build_windows": 1,
    "mc": 3,
    "reliability": 2,
}

interface_count_checks_630v18a = {
    name_630v18a:
        (
            parameter_counts_630v18a[
                name_630v18a
            ]
            ==
            expected_parameter_counts_630v18a[
                name_630v18a
            ]
        )
    for name_630v18a
    in expected_parameter_counts_630v18a
}

if not all(
    interface_count_checks_630v18a.values()
):
    failed_630v18a = [
        key
        for key, passed
        in interface_count_checks_630v18a.items()
        if not passed
    ]

    raise RuntimeError(
        "Live helper parameter-count mismatch: "
        + ", ".join(
            failed_630v18a
        )
    )


# =============================================================================
# 6) MODEL / DEVICE STATE — NO FORWARD PASS
# =============================================================================

model_630v18a = (
    MODEL_FD004_630I
)

if not isinstance(
    model_630v18a,
    torch.nn.Module,
):
    raise RuntimeError(
        "MODEL_FD004_630I is not a torch.nn.Module."
    )


parameter_count_630v18a = int(
    sum(
        parameter_630v18a.numel()
        for parameter_630v18a
        in model_630v18a.parameters()
    )
)

model_eval_630v18a = bool(
    not model_630v18a.training
)

try:
    model_device_630v18a = next(
        model_630v18a.parameters()
    ).device

except StopIteration as exc_630v18a:
    raise RuntimeError(
        "FD004 model has no parameters."
    ) from exc_630v18a


if parameter_count_630v18a != 19393:
    raise RuntimeError(
        "FD004 model parameter count changed."
    )

if not model_eval_630v18a:
    raise RuntimeError(
        "FD004 model is not in eval mode before admission."
    )


def _state_sha_630v18a(
    model_value_630v18a,
):

    h_630v18a = hashlib.sha256()

    state_630v18a = (
        model_value_630v18a.state_dict()
    )

    for key_630v18a in sorted(
        state_630v18a.keys()
    ):

        tensor_630v18a = (
            state_630v18a[
                key_630v18a
            ]
            .detach()
            .cpu()
            .contiguous()
        )

        arr_630v18a = (
            tensor_630v18a.numpy()
        )

        h_630v18a.update(
            key_630v18a.encode(
                "utf-8"
            )
        )

        h_630v18a.update(
            str(
                arr_630v18a.dtype
            ).encode(
                "utf-8"
            )
        )

        h_630v18a.update(
            str(
                tuple(
                    arr_630v18a.shape
                )
            ).encode(
                "utf-8"
            )
        )

        h_630v18a.update(
            arr_630v18a.tobytes(
                order="C"
            )
        )

    return h_630v18a.hexdigest()


model_state_before_630v18a = (
    _state_sha_630v18a(
        model_630v18a
    )
)


expected_restored_model_hash_630v18a = (
    "9078eaaa8e3c6568a1a8555fd0222155ec6e65e5c3c510b43c96c9d2f45d08f8"
)

resident_expected_model_hash_names_630v18a = tuple(
    sorted(
        name_630v18a
        for name_630v18a, value_630v18a
        in globals().items()
        if isinstance(
            value_630v18a,
            str,
        )
        and
        value_630v18a
        ==
        expected_restored_model_hash_630v18a
    )
)


# =============================================================================
# 7) FROZEN BASE IMMUTABILITY
# =============================================================================

base_before_630v18a = (
    FD004_DEV_CORRUPTION_BASE_630V17A
    .copy(deep=True)
)

base_shape_630v18a = tuple(
    FD004_DEV_CORRUPTION_BASE_630V17A.shape
)

base_has_regime_630v18a = bool(
    "regime_id"
    in
    FD004_DEV_CORRUPTION_BASE_630V17A.columns
)

base_regime_finite_630v18a = bool(
    np.isfinite(
        FD004_DEV_CORRUPTION_BASE_630V17A[
            "regime_id"
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)


base_unchanged_630v18a = bool(
    FD004_DEV_CORRUPTION_BASE_630V17A.equals(
        base_before_630v18a
    )
)

model_state_after_630v18a = (
    _state_sha_630v18a(
        model_630v18a
    )
)

model_unchanged_630v18a = bool(
    model_state_before_630v18a
    ==
    model_state_after_630v18a
)

if not (
    base_unchanged_630v18a
    and
    model_unchanged_630v18a
):
    raise RuntimeError(
        "Admission audit unexpectedly changed frozen state."
    )


# =============================================================================
# 8) ADMISSION CONTRACT
# =============================================================================

admission_checks_630v18a = {
    "all_frozen_hashes_exact":
        all(
            hash_checks_630v18a.values()
        ),

    "manifest_counts_exact":
        all(
            count_checks_630v18a.values()
        ),

    "manifest_identity_exact":
        all(
            identity_checks_630v18a.values()
        ),

    "family_sequence_exact":
        family_sequence_exact_630v18a,

    "family_blocks_exact":
        all(
            all(
                values_630v18a.values()
            )
            for values_630v18a
            in family_block_checks_630v18a.values()
        ),

    "inadmissible_family_counts_exact":
        inad_family_exact_630v18a,

    "routing_fields_complete":
        all(
            routing_field_checks_630v18a.values()
        ),

    "helper_parameter_counts_exact":
        all(
            interface_count_checks_630v18a.values()
        ),

    "model_parameter_count_exact":
        parameter_count_630v18a
        == 19393,

    "model_eval":
        model_eval_630v18a,

    "corruption_base_shape_exact":
        base_shape_630v18a
        ==
        (
            12274,
            27,
        ),

    "corruption_base_has_regime_id":
        base_has_regime_630v18a,

    "corruption_base_regime_finite":
        base_regime_finite_630v18a,

    "corruption_base_unchanged":
        base_unchanged_630v18a,

    "model_unchanged":
        model_unchanged_630v18a,
}


if not all(
    admission_checks_630v18a.values()
):
    failed_630v18a = [
        key
        for key, passed
        in admission_checks_630v18a.items()
        if not passed
    ]

    raise RuntimeError(
        "381-scenario admission check failed: "
        + ", ".join(
            failed_630v18a
        )
    )


FD004_381_EXECUTION_ADMISSION_CONTRACT_630V18A = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18A",

    "status":
        "381_SCENARIO_EXECUTION_ADMITTED_NOT_STARTED",

    "scientific_manifest_count":
        441,

    "execution_manifest_count":
        381,

    "inadmissible_manifest_count":
        60,

    "family_order":
        family_order_630v18a,

    "family_block_spec":
        expected_block_spec_630v18a,

    "inadmissible_family_counts":
        {
            "bias": 24,
            "noise": 12,
            "drift": 24,
            "dropout": 0,
            "stuck": 0,
        },

    "signatures":
        signatures_630v18a,

    "parameter_counts":
        parameter_counts_630v18a,

    "model_parameter_count":
        parameter_count_630v18a,

    "model_device":
        str(
            model_device_630v18a
        ),

    "model_eval":
        model_eval_630v18a,

    "resident_frozen_model_hash_authority_names":
        resident_expected_model_hash_names_630v18a,

    "admission_checks":
        admission_checks_630v18a,

    "execution_retention_rule":
        (
            "retain per-scenario deterministic prediction + "
            "MC STD + final risk; hash full 50-pass MC then discard"
        ),

    "dev_outcomes_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "scenario_execution_started":
        False,
}


FD004_381_EXECUTION_ADMISSION_SHA256_630V18A = (
    hashlib.sha256(
        json.dumps(
            FD004_381_EXECUTION_ADMISSION_CONTRACT_630V18A,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 9) LOCK FLAGS
# =============================================================================

FD004_381_EXECUTION_ADMITTED_630V18A = True
FD004_381_EXECUTION_STARTED_630V18A = False

FD004_CORRUPTION_EXECUTED_630V18A = False
FD004_PREPROCESSING_EXECUTED_630V18A = False
FD004_MODEL_INFERENCE_PERFORMED_630V18A = False
FD004_MC_EXECUTION_PERFORMED_630V18A = False

FD004_DEV_TARGETS_ERRORS_USED_630V18A = False
FD004_THRESHOLD_SELECTED_630V18A = False
FD004_POLICY_SELECTED_630V18A = False

FD004_TEST_CONTENTS_READ_630V18A = False
FD004_RUL_CONTENTS_READ_630V18A = False

all_checks_passed_630v18a = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18A — FD004 ZERO-EXECUTION "
    "381-SCENARIO ADMISSION AUDIT"
)

print("=" * 176)

print("Scenario execution started        :", False)
print("Corruption executed               :", False)
print("Preprocessing executed            :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- FROZEN EVIDENCE HASHES ---")

for key_630v18a in expected_hashes_630v18a:
    print(
        f"{key_630v18a:32s}:",
        hash_checks_630v18a[
            key_630v18a
        ],
    )


print("\n--- MANIFEST POPULATIONS ---")

print(
    "Scientific / annotated            :",
    len(
        annotated_630v18a
    ),
)

print(
    "Executable                        :",
    len(
        exec_manifest_630v18a
    ),
)

print(
    "Structurally inadmissible         :",
    len(
        inadmissible_630v18a
    ),
)

print(
    "Partition exact                   :",
    count_checks_630v18a[
        "partition_total_441"
    ],
)

print(
    "Identity partition exact          :",
    identity_checks_630v18a[
        "partition_matches_annotated"
    ],
)


print("\n--- EXECUTABLE FAMILY BLOCKS ---")

for family_630v18a in family_order_630v18a:

    rows_630v18a = (
        exec_manifest_630v18a.loc[
            exec_manifest_630v18a[
                "family"
            ].astype(str)
            ==
            family_630v18a
        ]
    )

    nums_630v18a = (
        rows_630v18a[
            "scenario_number"
        ].to_numpy(
            dtype=np.int64
        )
    )

    print(
        f"{family_630v18a:8s}: "
        f"count={len(rows_630v18a):3d}, "
        f"scenario_range=("
        f"{int(nums_630v18a.min())}, "
        f"{int(nums_630v18a.max())}), "
        f"checks="
        f"{all(family_block_checks_630v18a[family_630v18a].values())}"
    )


print("\n--- STRUCTURALLY INADMISSIBLE ---")

print(
    "Family counts                     :",
    inad_family_counts_630v18a,
)

print(
    "Counts exact                      :",
    inad_family_exact_630v18a,
)


print("\n--- LIVE HELPER INTERFACES (NOT CALLED) ---")

for name_630v18a in (
    "dispatcher",
    "normalize",
    "build_windows",
    "mc",
    "reliability",
):
    print(
        f"{name_630v18a:14s}: "
        f"{signatures_630v18a[name_630v18a]}"
    )


print("\n--- MODEL / BASE STATE ---")

print(
    "Model parameter count             :",
    parameter_count_630v18a,
)

print(
    "Model device                      :",
    model_device_630v18a,
)

print(
    "Model eval mode                   :",
    model_eval_630v18a,
)

print(
    "Frozen model hash authority names :",
    resident_expected_model_hash_names_630v18a,
)

print(
    "Model unchanged                   :",
    model_unchanged_630v18a,
)

print(
    "Corruption base shape             :",
    base_shape_630v18a,
)

print(
    "Corruption base has regime_id     :",
    base_has_regime_630v18a,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18a,
)


print("\n--- ADMISSION ---")

print(
    "All admission checks              :",
    all(
        admission_checks_630v18a.values()
    ),
)

print(
    "381-scenario execution admitted   :",
    FD004_381_EXECUTION_ADMITTED_630V18A,
)

print(
    "381-scenario execution started    :",
    FD004_381_EXECUTION_STARTED_630V18A,
)

print(
    "Admission SHA-256                 :",
    FD004_381_EXECUTION_ADMISSION_SHA256_630V18A,
)

print(
    "\nPASS — the frozen 381-scenario FD004 controlled-corruption "
    "execution is admitted, but no scenario has been executed "
    "in this step."
)

print(
    "Next: execute only the first frozen family block "
    "(bias, 102 admissible scenarios), retain the frozen evidence "
    "products, and checkpoint that block before proceeding."
)

In [ ]:
# STEP 6.30V-18B0 — FD004 NORMALIZATION-BRIDGE INPUT CONTRACT SOURCE AUDIT
#
# ZERO EXECUTION:
# - no normalizer call
# - no corruption
# - no model inference
# - no MC
# - no DEV targets/errors
# - no TEST / RUL

import ast
import inspect
import hashlib
import json
import dis
import io
import contextlib


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18b0 = (
    "all_checks_passed_630v18a",
    "normalize_fd004_raw_630v2",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
)

missing_630v18b0 = [
    name_630v18b0
    for name_630v18b0 in required_630v18b0
    if name_630v18b0 not in globals()
]

if missing_630v18b0:
    raise RuntimeError(
        "Missing STEP 6.30V-18B0 authority object(s): "
        + ", ".join(missing_630v18b0)
    )

if not bool(all_checks_passed_630v18a):
    raise RuntimeError(
        "STEP 6.30V-18A must remain passed."
    )


# =============================================================================
# 1) LIVE FUNCTION AUTHORITY — INSPECTION ONLY
# =============================================================================

fn_630v18b0 = normalize_fd004_raw_630v2

if not callable(fn_630v18b0):
    raise RuntimeError(
        "normalize_fd004_raw_630v2 is not callable."
    )

signature_630v18b0 = str(
    inspect.signature(
        fn_630v18b0
    )
)


# =============================================================================
# 2) RECOVER FUNCTION SOURCE WITHOUT EXECUTING IT
# =============================================================================

source_630v18b0 = None
source_origin_630v18b0 = None

try:
    candidate_630v18b0 = inspect.getsource(
        fn_630v18b0
    )

    if (
        isinstance(
            candidate_630v18b0,
            str,
        )
        and
        "normalize_fd004_raw_630v2"
        in candidate_630v18b0
    ):
        source_630v18b0 = (
            candidate_630v18b0
        )

        source_origin_630v18b0 = (
            "inspect.getsource"
        )

except Exception:
    pass


# Conservative fallback:
# inspect resident STRING globals only.
if source_630v18b0 is None:

    string_hits_630v18b0 = []

    for (
        name_630v18b0,
        value_630v18b0,
    ) in list(
        globals().items()
    ):

        if not isinstance(
            value_630v18b0,
            str,
        ):
            continue

        if (
            "normalize_fd004_raw_630v2"
            not in value_630v18b0
        ):
            continue

        try:
            tree_630v18b0 = ast.parse(
                value_630v18b0
            )
        except Exception:
            continue

        for node_630v18b0 in ast.walk(
            tree_630v18b0
        ):

            if (
                isinstance(
                    node_630v18b0,
                    ast.FunctionDef,
                )
                and
                node_630v18b0.name
                ==
                "normalize_fd004_raw_630v2"
            ):

                lines_630v18b0 = (
                    value_630v18b0.splitlines(
                        keepends=True
                    )
                )

                extracted_630v18b0 = "".join(
                    lines_630v18b0[
                        node_630v18b0.lineno - 1:
                        node_630v18b0.end_lineno
                    ]
                )

                string_hits_630v18b0.append(
                    (
                        name_630v18b0,
                        extracted_630v18b0,
                    )
                )

    unique_sources_630v18b0 = {}

    for (
        name_630v18b0,
        text_630v18b0,
    ) in string_hits_630v18b0:

        sha_630v18b0 = hashlib.sha256(
            text_630v18b0.encode(
                "utf-8"
            )
        ).hexdigest()

        unique_sources_630v18b0.setdefault(
            sha_630v18b0,
            [],
        ).append(
            (
                name_630v18b0,
                text_630v18b0,
            )
        )

    if len(
        unique_sources_630v18b0
    ) == 1:

        only_sha_630v18b0 = next(
            iter(
                unique_sources_630v18b0
            )
        )

        (
            source_name_630v18b0,
            source_630v18b0,
        ) = unique_sources_630v18b0[
            only_sha_630v18b0
        ][0]

        source_origin_630v18b0 = (
            "resident_string_global:"
            + source_name_630v18b0
        )


# =============================================================================
# 3) BYTECODE EVIDENCE — ALSO ZERO EXECUTION
# =============================================================================

bytecode_buffer_630v18b0 = (
    io.StringIO()
)

with contextlib.redirect_stdout(
    bytecode_buffer_630v18b0
):
    dis.dis(
        fn_630v18b0
    )

bytecode_text_630v18b0 = (
    bytecode_buffer_630v18b0.getvalue()
)

code_obj_630v18b0 = (
    fn_630v18b0.__code__
)

code_names_630v18b0 = tuple(
    code_obj_630v18b0.co_names
)

code_string_constants_630v18b0 = tuple(
    value_630v18b0
    for value_630v18b0
    in code_obj_630v18b0.co_consts
    if isinstance(
        value_630v18b0,
        str,
    )
)


# =============================================================================
# 4) INPUT-SHAPE / COLUMN AUTHORITY
# =============================================================================

base_columns_630v18b0 = tuple(
    FD004_DEV_CORRUPTION_BASE_630V17A.columns
)

raw_columns_630v18b0 = tuple(
    col_630v18b0
    for col_630v18b0
    in base_columns_630v18b0
    if col_630v18b0
    != "regime_id"
)

source_found_630v18b0 = isinstance(
    source_630v18b0,
    str,
)

source_sha_630v18b0 = (
    hashlib.sha256(
        source_630v18b0.encode(
            "utf-8"
        )
    ).hexdigest()
    if source_found_630v18b0
    else None
)

source_mentions_regime_id_630v18b0 = (
    "regime_id"
    in source_630v18b0
    if source_found_630v18b0
    else None
)

bytecode_mentions_regime_id_630v18b0 = bool(
    "regime_id"
    in code_names_630v18b0
    or
    "regime_id"
    in code_string_constants_630v18b0
    or
    "regime_id"
    in bytecode_text_630v18b0
)


# =============================================================================
# 5) LOCK AUDIT
# =============================================================================

FD004_NORMALIZER_SOURCE_630V18B0 = (
    source_630v18b0
)

FD004_NORMALIZER_BYTECODE_630V18B0 = (
    bytecode_text_630v18b0
)

FD004_NORMALIZER_INPUT_CONTRACT_AUDIT_630V18B0 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18B0",

    "signature":
        signature_630v18b0,

    "source_found":
        source_found_630v18b0,

    "source_origin":
        source_origin_630v18b0,

    "source_sha256":
        source_sha_630v18b0,

    "source_mentions_regime_id":
        source_mentions_regime_id_630v18b0,

    "bytecode_mentions_regime_id":
        bytecode_mentions_regime_id_630v18b0,

    "execution_base_column_count":
        int(
            len(
                base_columns_630v18b0
            )
        ),

    "raw_column_count_without_regime_id":
        int(
            len(
                raw_columns_630v18b0
            )
        ),

    "source_executed":
        False,

    "normalizer_executed":
        False,

    "corruption_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_NORMALIZER_INPUT_CONTRACT_AUDIT_SHA256_630V18B0 = (
    hashlib.sha256(
        json.dumps(
            FD004_NORMALIZER_INPUT_CONTRACT_AUDIT_630V18B0,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_NORMALIZER_EXECUTED_630V18B0 = False
FD004_CORRUPTION_EXECUTED_630V18B0 = False
FD004_MODEL_INFERENCE_PERFORMED_630V18B0 = False
FD004_MC_EXECUTION_PERFORMED_630V18B0 = False
FD004_DEV_TARGETS_ERRORS_USED_630V18B0 = False
FD004_TEST_CONTENTS_READ_630V18B0 = False
FD004_RUL_CONTENTS_READ_630V18B0 = False

all_checks_passed_630v18b0 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18B0 — FD004 NORMALIZATION-BRIDGE "
    "INPUT CONTRACT SOURCE AUDIT"
)

print("=" * 176)

print("Normalizer executed               :", False)
print("Corruption executed               :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- LIVE NORMALIZER ---")

print(
    "Signature                         :",
    signature_630v18b0,
)

print(
    "Source recovered                  :",
    source_found_630v18b0,
)

print(
    "Source origin                     :",
    source_origin_630v18b0,
)

print(
    "Source SHA-256                    :",
    source_sha_630v18b0,
)

print(
    "Source mentions regime_id         :",
    source_mentions_regime_id_630v18b0,
)

print(
    "Bytecode mentions regime_id       :",
    bytecode_mentions_regime_id_630v18b0,
)

print(
    "Execution base column count       :",
    len(
        base_columns_630v18b0
    ),
)

print(
    "Raw columns excluding regime_id   :",
    len(
        raw_columns_630v18b0
    ),
)


if source_found_630v18b0:

    print(
        "\n--- EXACT NORMALIZER SOURCE ---"
    )

    for (
        line_number_630v18b0,
        line_text_630v18b0,
    ) in enumerate(
        source_630v18b0.splitlines(),
        start=1,
    ):

        print(
            f"{line_number_630v18b0:03d} | "
            f"{line_text_630v18b0}"
        )

else:

    print(
        "\n--- BYTECODE FALLBACK ---"
    )

    print(
        bytecode_text_630v18b0
    )


print("\n--- AUDIT FINGERPRINT ---")

print(
    "Normalizer input audit SHA-256    :",
    FD004_NORMALIZER_INPUT_CONTRACT_AUDIT_SHA256_630V18B0,
)

print(
    "\nPASS — the live FD004 normalization helper was inspected "
    "without executing it."
)

print(
    "Next: choose the exact dispatcher-output projection from this "
    "evidence before starting the bias block."
)

In [ ]:
# STEP 6.30V-18B1 — FD004 DETERMINISTIC-PREDICTION AUTHORITY LOCATOR
#
# ZERO EXECUTION:
# - no normalizer call
# - no model forward pass
# - no corruption / MC / reliability call
# - no DEV targets/errors
# - no TEST / RUL

import inspect
import hashlib
import json
from collections.abc import Mapping

required_630v18b1 = (
    "all_checks_passed_630v18b0",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6",
    "MODEL_FD004_630I",
)

missing_630v18b1 = [
    name for name in required_630v18b1
    if name not in globals()
]

if missing_630v18b1:
    raise RuntimeError(
        "Missing STEP 6.30V-18B1 authority object(s): "
        + ", ".join(missing_630v18b1)
    )

if not bool(all_checks_passed_630v18b0):
    raise RuntimeError(
        "STEP 6.30V-18B0 must remain passed."
    )

EXPECTED_EXECUTION_CONTRACT_SHA256_630V18B1 = (
    "687f663209e062f0da2827911ddc38ee185c4d0e181e0c1dee6ccc7c5760e21a"
)

if (
    str(FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6)
    != EXPECTED_EXECUTION_CONTRACT_SHA256_630V18B1
):
    raise RuntimeError(
        "Frozen full-execution contract fingerprint changed."
    )

# =============================================================================
# 1) DISCOVER RESIDENT FULL-EXECUTION CONTRACT OBJECTS
# =============================================================================

contract_candidates_630v18b1 = {}

for name_630v18b1, value_630v18b1 in list(globals().items()):
    upper_630v18b1 = str(name_630v18b1).upper()

    if (
        "FD004" in upper_630v18b1
        and "FULL" in upper_630v18b1
        and "CORRUPTION" in upper_630v18b1
        and "EXECUTION" in upper_630v18b1
        and "CONTRACT" in upper_630v18b1
        and "SHA256" not in upper_630v18b1
    ):
        contract_candidates_630v18b1[
            str(name_630v18b1)
        ] = value_630v18b1

contract_descriptions_630v18b1 = {}

for name_630v18b1, value_630v18b1 in contract_candidates_630v18b1.items():

    if isinstance(
        value_630v18b1,
        Mapping,
    ):
        keys_630v18b1 = tuple(
            str(k)
            for k in value_630v18b1.keys()
        )

        safe_items_630v18b1 = {}

        for key_630v18b1, item_630v18b1 in value_630v18b1.items():

            key_text_630v18b1 = str(
                key_630v18b1
            )

            if isinstance(
                item_630v18b1,
                (
                    str,
                    int,
                    float,
                    bool,
                    type(None),
                    tuple,
                    list,
                ),
            ):
                safe_items_630v18b1[
                    key_text_630v18b1
                ] = repr(
                    item_630v18b1
                )

            elif callable(
                item_630v18b1
            ):
                try:
                    sig_630v18b1 = str(
                        inspect.signature(
                            item_630v18b1
                        )
                    )
                except Exception:
                    sig_630v18b1 = None

                safe_items_630v18b1[
                    key_text_630v18b1
                ] = (
                    f"<callable "
                    f"{getattr(item_630v18b1, '__name__', type(item_630v18b1).__name__)} "
                    f"signature={sig_630v18b1}>"
                )

            else:
                safe_items_630v18b1[
                    key_text_630v18b1
                ] = (
                    f"<{type(item_630v18b1).__name__}>"
                )

        contract_descriptions_630v18b1[
            name_630v18b1
        ] = {
            "type":
                type(
                    value_630v18b1
                ).__name__,

            "keys":
                keys_630v18b1,

            "safe_items":
                safe_items_630v18b1,
        }

    else:
        contract_descriptions_630v18b1[
            name_630v18b1
        ] = {
            "type":
                type(
                    value_630v18b1
                ).__name__,

            "repr":
                repr(
                    value_630v18b1
                )[:2000],
        }


# =============================================================================
# 2) DISCOVER LIVE PREDICTION CALLABLES — DO NOT CALL
# =============================================================================

predictor_candidates_630v18b1 = {}

for name_630v18b1, value_630v18b1 in list(globals().items()):

    lower_630v18b1 = str(
        name_630v18b1
    ).lower()

    if not callable(
        value_630v18b1
    ):
        continue

    if "predict" not in lower_630v18b1:
        continue

    if not (
        "fd004" in lower_630v18b1
        or
        "630" in lower_630v18b1
    ):
        continue

    try:
        signature_630v18b1 = str(
            inspect.signature(
                value_630v18b1
            )
        )
    except Exception:
        signature_630v18b1 = None

    source_630v18b1 = None
    source_sha_630v18b1 = None

    try:
        source_630v18b1 = inspect.getsource(
            value_630v18b1
        )

        source_sha_630v18b1 = hashlib.sha256(
            source_630v18b1.encode(
                "utf-8"
            )
        ).hexdigest()

    except Exception:
        pass

    predictor_candidates_630v18b1[
        str(name_630v18b1)
    ] = {
        "object":
            value_630v18b1,

        "signature":
            signature_630v18b1,

        "source":
            source_630v18b1,

        "source_sha256":
            source_sha_630v18b1,
    }


# =============================================================================
# 3) DISCOVER RESIDENT FD004 PREDICTION EVIDENCE OBJECTS
# =============================================================================

prediction_object_descriptions_630v18b1 = {}

for name_630v18b1, value_630v18b1 in list(globals().items()):

    upper_630v18b1 = str(
        name_630v18b1
    ).upper()

    if "FD004" not in upper_630v18b1:
        continue

    if not any(
        token_630v18b1 in upper_630v18b1
        for token_630v18b1 in (
            "PRED",
            "PREDICTION",
            "DETERMINISTIC",
        )
    ):
        continue

    if callable(
        value_630v18b1
    ):
        continue

    shape_630v18b1 = getattr(
        value_630v18b1,
        "shape",
        None,
    )

    prediction_object_descriptions_630v18b1[
        str(name_630v18b1)
    ] = {
        "type":
            type(
                value_630v18b1
            ).__name__,

        "shape":
            (
                tuple(
                    shape_630v18b1
                )
                if shape_630v18b1 is not None
                else None
            ),
    }


# =============================================================================
# 4) MODEL STATE METADATA ONLY
# =============================================================================

model_630v18b1 = (
    MODEL_FD004_630I
)

model_eval_630v18b1 = bool(
    not model_630v18b1.training
)

model_device_630v18b1 = str(
    next(
        model_630v18b1.parameters()
    ).device
)


# =============================================================================
# 5) LOCK LOCATOR EVIDENCE
# =============================================================================

FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_630V18B1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18B1",

    "frozen_execution_contract_sha256":
        str(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_SHA256_630V6
        ),

    "contract_candidates":
        contract_descriptions_630v18b1,

    "predictor_candidates":
        {
            name_630v18b1: {
                "signature":
                    item_630v18b1[
                        "signature"
                    ],

                "source_sha256":
                    item_630v18b1[
                        "source_sha256"
                    ],
            }
            for name_630v18b1, item_630v18b1
            in predictor_candidates_630v18b1.items()
        },

    "prediction_object_descriptions":
        prediction_object_descriptions_630v18b1,

    "model_eval":
        model_eval_630v18b1,

    "model_device":
        model_device_630v18b1,

    "normalizer_direct_27_column_input_admitted":
        True,

    "normalizer_reason":
        (
            "normalize_fd004_raw_630v2 explicitly selects only frozen "
            "operating-setting and sensor columns and receives regime IDs "
            "through its separate argument; extra regime_id column is not "
            "part of the normalized feature matrix"
        ),

    "source_executed":
        False,

    "model_forward_performed":
        False,

    "corruption_executed":
        False,

    "mc_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_SHA256_630V18B1 = (
    hashlib.sha256(
        json.dumps(
            FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_630V18B1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_MODEL_FORWARD_PERFORMED_630V18B1 = False
FD004_CORRUPTION_EXECUTED_630V18B1 = False
FD004_MC_EXECUTION_PERFORMED_630V18B1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V18B1 = False
FD004_TEST_CONTENTS_READ_630V18B1 = False
FD004_RUL_CONTENTS_READ_630V18B1 = False

all_checks_passed_630v18b1 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18B1 — FD004 DETERMINISTIC-PREDICTION "
    "AUTHORITY LOCATOR"
)

print("=" * 176)

print("Model forward performed           :", False)
print("Corruption executed               :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- NORMALIZER INPUT DECISION ---")

print(
    "27-column dispatcher output direct:",
    True,
)

print(
    "Reason                            :",
    FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_630V18B1[
        "normalizer_reason"
    ],
)

print(
    "\n--- FULL-EXECUTION CONTRACT OBJECT CANDIDATES ---"
)

if contract_descriptions_630v18b1:

    for name_630v18b1, desc_630v18b1 in sorted(
        contract_descriptions_630v18b1.items()
    ):

        print(
            f"\n{name_630v18b1}"
        )

        print(
            "  type                             :",
            desc_630v18b1[
                "type"
            ],
        )

        if "keys" in desc_630v18b1:

            print(
                "  keys                             :",
                desc_630v18b1[
                    "keys"
                ],
            )

            for (
                key_630v18b1,
                value_630v18b1,
            ) in desc_630v18b1[
                "safe_items"
            ].items():

                print(
                    f"  {key_630v18b1:<32s}:",
                    value_630v18b1,
                )

        else:

            print(
                "  repr                             :",
                desc_630v18b1[
                    "repr"
                ],
            )

else:

    print(
        "No resident full-execution contract object discovered."
    )

print(
    "\n--- LIVE PREDICTOR CALLABLE CANDIDATES ---"
)

if predictor_candidates_630v18b1:

    for name_630v18b1, item_630v18b1 in sorted(
        predictor_candidates_630v18b1.items()
    ):

        print(
            f"\n{name_630v18b1}"
        )

        print(
            "  signature                        :",
            item_630v18b1[
                "signature"
            ],
        )

        print(
            "  source SHA-256                   :",
            item_630v18b1[
                "source_sha256"
            ],
        )

        if item_630v18b1[
            "source"
        ] is not None:

            print(
                "  source:"
            )

            for (
                line_no_630v18b1,
                line_630v18b1,
            ) in enumerate(
                item_630v18b1[
                    "source"
                ].splitlines(),
                start=1,
            ):

                print(
                    f"    {line_no_630v18b1:03d} | "
                    f"{line_630v18b1}"
                )

else:

    print(
        "No live FD004/630 prediction callable discovered."
    )

print(
    "\n--- RESIDENT FD004 PREDICTION EVIDENCE OBJECTS ---"
)

for name_630v18b1, item_630v18b1 in sorted(
    prediction_object_descriptions_630v18b1.items()
):

    print(
        f"{name_630v18b1:<52s}: "
        f"type={item_630v18b1['type']}, "
        f"shape={item_630v18b1['shape']}"
    )

print("\n--- MODEL STATE ---")

print(
    "Model eval mode                   :",
    model_eval_630v18b1,
)

print(
    "Model device                      :",
    model_device_630v18b1,
)

print("\n--- AUDIT FINGERPRINT ---")

print(
    "Prediction-authority locator SHA  :",
    FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_SHA256_630V18B1,
)

print(
    "\nPASS — deterministic-prediction authorities were inspected "
    "without running the model."
)

print(
    "Next: freeze the exact deterministic inference path from this "
    "evidence, then start only the 102-scenario bias block."
)

In [ ]:
# STEP 6.30V-18B2 — FD004 EXACT DETERMINISTIC-INFERENCE SOURCE RECOVERY
#
# SOURCE/HISTORY INSPECTION ONLY.
# NO model forward pass.
# NO corruption.
# NO normalization/window execution.
# NO MC / reliability execution.
# NO DEV targets/errors.
# NO TEST / RUL.

import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18b2 = (
    "all_checks_passed_630v18b1",
    "FD004_DETERMINISTIC_PREDICTION_AUTHORITY_LOCATOR_SHA256_630V18B1",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6",
)

missing_630v18b2 = [
    name_630v18b2
    for name_630v18b2 in required_630v18b2
    if name_630v18b2 not in globals()
]

if missing_630v18b2:
    raise RuntimeError(
        "Missing STEP 6.30V-18B2 authority object(s): "
        + ", ".join(missing_630v18b2)
    )

if not bool(all_checks_passed_630v18b1):
    raise RuntimeError(
        "STEP 6.30V-18B1 must remain passed."
    )


# =============================================================================
# 1) TARGET ALREADY-EXECUTED DETERMINISTIC EVIDENCE
# =============================================================================

target_symbols_630v18b2 = (
    "FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2",
    "FD004_FIT_DETERMINISTIC_PREDICTION_630O",
    "FD004_CLEAN_DEV_PREDICTION_630K",
)

primary_target_630v18b2 = (
    "FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2"
)


# =============================================================================
# 2) COLLECT SOURCE TEXTS WITHOUT EXECUTING ANYTHING
# =============================================================================

source_records_630v18b2 = []


def _add_source_record_630v18b2(
    origin_630v18b2,
    text_630v18b2,
):
    if not isinstance(
        text_630v18b2,
        str,
    ):
        return

    if not any(
        target_630v18b2 in text_630v18b2
        for target_630v18b2
        in target_symbols_630v18b2
    ):
        return

    sha_630v18b2 = hashlib.sha256(
        text_630v18b2.encode("utf-8")
    ).hexdigest()

    source_records_630v18b2.append(
        {
            "origin":
                str(origin_630v18b2),

            "text":
                text_630v18b2,

            "sha256":
                sha_630v18b2,

            "length":
                len(text_630v18b2),

            "targets":
                tuple(
                    target_630v18b2
                    for target_630v18b2
                    in target_symbols_630v18b2
                    if target_630v18b2
                    in text_630v18b2
                ),
        }
    )


# IPython input history is the most direct notebook-source authority.
if (
    "In" in globals()
    and
    isinstance(
        globals()["In"],
        list,
    )
):
    for idx_630v18b2, text_630v18b2 in enumerate(
        globals()["In"]
    ):
        _add_source_record_630v18b2(
            f"In[{idx_630v18b2}]",
            text_630v18b2,
        )


# Also inspect resident source/cell/history text containers.
candidate_name_tokens_630v18b2 = (
    "source",
    "cell",
    "history",
    "630v",
    "630o",
    "630k",
)

visited_ids_630v18b2 = set()
node_count_630v18b2 = 0
MAX_NODES_630V18B2 = 30000
MAX_DEPTH_630V18B2 = 5


def _walk_source_container_630v18b2(
    value_630v18b2,
    path_630v18b2,
    depth_630v18b2,
):
    global node_count_630v18b2

    if node_count_630v18b2 >= MAX_NODES_630V18B2:
        return

    node_count_630v18b2 += 1

    if isinstance(
        value_630v18b2,
        str,
    ):
        _add_source_record_630v18b2(
            path_630v18b2,
            value_630v18b2,
        )
        return

    if depth_630v18b2 >= MAX_DEPTH_630V18B2:
        return

    if not isinstance(
        value_630v18b2,
        (
            list,
            tuple,
            dict,
        ),
    ):
        return

    object_id_630v18b2 = id(
        value_630v18b2
    )

    if object_id_630v18b2 in visited_ids_630v18b2:
        return

    visited_ids_630v18b2.add(
        object_id_630v18b2
    )

    if isinstance(
        value_630v18b2,
        dict,
    ):
        iterator_630v18b2 = (
            value_630v18b2.items()
        )

        for (
            key_630v18b2,
            child_630v18b2,
        ) in iterator_630v18b2:

            _walk_source_container_630v18b2(
                child_630v18b2,
                (
                    f"{path_630v18b2}"
                    f"[{key_630v18b2!r}]"
                ),
                depth_630v18b2 + 1,
            )

    else:

        for idx_630v18b2, child_630v18b2 in enumerate(
            value_630v18b2
        ):
            _walk_source_container_630v18b2(
                child_630v18b2,
                (
                    f"{path_630v18b2}"
                    f"[{idx_630v18b2}]"
                ),
                depth_630v18b2 + 1,
            )


for (
    name_630v18b2,
    value_630v18b2,
) in list(
    globals().items()
):

    lower_name_630v18b2 = str(
        name_630v18b2
    ).lower()

    if not any(
        token_630v18b2
        in lower_name_630v18b2
        for token_630v18b2
        in candidate_name_tokens_630v18b2
    ):
        continue

    if name_630v18b2 == "In":
        continue

    _walk_source_container_630v18b2(
        value_630v18b2,
        str(name_630v18b2),
        0,
    )


# =============================================================================
# 3) DEDUPLICATE IDENTICAL SOURCE TEXTS
# =============================================================================

dedup_630v18b2 = {}

for record_630v18b2 in source_records_630v18b2:

    sha_630v18b2 = (
        record_630v18b2[
            "sha256"
        ]
    )

    if sha_630v18b2 not in dedup_630v18b2:

        dedup_630v18b2[
            sha_630v18b2
        ] = {
            "sha256":
                sha_630v18b2,

            "text":
                record_630v18b2[
                    "text"
                ],

            "length":
                record_630v18b2[
                    "length"
                ],

            "targets":
                set(
                    record_630v18b2[
                        "targets"
                    ]
                ),

            "origins":
                [
                    record_630v18b2[
                        "origin"
                    ]
                ],
        }

    else:

        dedup_630v18b2[
            sha_630v18b2
        ][
            "targets"
        ].update(
            record_630v18b2[
                "targets"
            ]
        )

        dedup_630v18b2[
            sha_630v18b2
        ][
            "origins"
        ].append(
            record_630v18b2[
                "origin"
            ]
        )


unique_records_630v18b2 = []

for item_630v18b2 in dedup_630v18b2.values():

    unique_records_630v18b2.append(
        {
            "sha256":
                item_630v18b2[
                    "sha256"
                ],

            "text":
                item_630v18b2[
                    "text"
                ],

            "length":
                item_630v18b2[
                    "length"
                ],

            "targets":
                tuple(
                    sorted(
                        item_630v18b2[
                            "targets"
                        ]
                    )
                ),

            "origins":
                tuple(
                    sorted(
                        set(
                            item_630v18b2[
                                "origins"
                            ]
                        )
                    )
                ),
        }
    )

unique_records_630v18b2 = tuple(
    sorted(
        unique_records_630v18b2,
        key=lambda item_630v18b2: (
            primary_target_630v18b2
            not in item_630v18b2[
                "targets"
            ],
            item_630v18b2[
                "length"
            ],
            item_630v18b2[
                "sha256"
            ],
        ),
    )
)


# =============================================================================
# 4) EXTRACT LOCAL CONTEXT AROUND EACH TARGET
# =============================================================================

context_records_630v18b2 = []

for source_record_630v18b2 in unique_records_630v18b2:

    lines_630v18b2 = (
        source_record_630v18b2[
            "text"
        ].splitlines()
    )

    for target_630v18b2 in (
        source_record_630v18b2[
            "targets"
        ]
    ):

        matched_lines_630v18b2 = [
            line_number_630v18b2
            for (
                line_number_630v18b2,
                line_text_630v18b2,
            ) in enumerate(
                lines_630v18b2,
                start=1,
            )
            if target_630v18b2
            in line_text_630v18b2
        ]

        for matched_line_630v18b2 in matched_lines_630v18b2:

            # Wide source window so the batch loop and assignment are visible.
            start_630v18b2 = max(
                1,
                matched_line_630v18b2 - 100,
            )

            end_630v18b2 = min(
                len(
                    lines_630v18b2
                ),
                matched_line_630v18b2 + 70,
            )

            context_630v18b2 = "\n".join(
                (
                    f"{line_number_630v18b2:04d} | "
                    f"{lines_630v18b2[line_number_630v18b2 - 1]}"
                )
                for line_number_630v18b2
                in range(
                    start_630v18b2,
                    end_630v18b2 + 1,
                )
            )

            context_records_630v18b2.append(
                {
                    "target":
                        target_630v18b2,

                    "source_sha256":
                        source_record_630v18b2[
                            "sha256"
                        ],

                    "source_length":
                        source_record_630v18b2[
                            "length"
                        ],

                    "origins":
                        source_record_630v18b2[
                            "origins"
                        ],

                    "matched_line":
                        int(
                            matched_line_630v18b2
                        ),

                    "context_start":
                        int(
                            start_630v18b2
                        ),

                    "context_end":
                        int(
                            end_630v18b2
                        ),

                    "context":
                        context_630v18b2,
                }
            )


primary_contexts_630v18b2 = tuple(
    record_630v18b2
    for record_630v18b2
    in context_records_630v18b2
    if record_630v18b2[
        "target"
    ]
    ==
    primary_target_630v18b2
)


# =============================================================================
# 5) LOCK SOURCE-RECOVERY EVIDENCE
# =============================================================================

FD004_DETERMINISTIC_SOURCE_RECORDS_630V18B2 = (
    unique_records_630v18b2
)

FD004_DETERMINISTIC_SOURCE_CONTEXTS_630V18B2 = (
    tuple(
        context_records_630v18b2
    )
)

FD004_SMOKE_DETERMINISTIC_SOURCE_CONTEXTS_630V18B2 = (
    primary_contexts_630v18b2
)


audit_payload_630v18b2 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18B2",

    "primary_target":
        primary_target_630v18b2,

    "target_symbols":
        target_symbols_630v18b2,

    "raw_source_hit_count":
        int(
            len(
                source_records_630v18b2
            )
        ),

    "unique_source_count":
        int(
            len(
                unique_records_630v18b2
            )
        ),

    "primary_context_count":
        int(
            len(
                primary_contexts_630v18b2
            )
        ),

    "source_summaries":
        tuple(
            {
                "sha256":
                    item_630v18b2[
                        "sha256"
                    ],

                "length":
                    int(
                        item_630v18b2[
                            "length"
                        ]
                    ),

                "targets":
                    item_630v18b2[
                        "targets"
                    ],

                "origins":
                    item_630v18b2[
                        "origins"
                    ],
            }
            for item_630v18b2
            in unique_records_630v18b2
        ),

    "frozen_deterministic_batch_size":
        int(
            FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6[
                "deterministic_eval_batch_size"
            ]
        ),

    "model_forward_performed":
        False,

    "corruption_executed":
        False,

    "normalization_executed":
        False,

    "window_build_executed":
        False,

    "mc_execution_performed":
        False,

    "reliability_executed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DETERMINISTIC_SOURCE_RECOVERY_AUDIT_630V18B2 = (
    audit_payload_630v18b2
)

FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630v18b2,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_MODEL_FORWARD_PERFORMED_630V18B2 = False
FD004_CORRUPTION_EXECUTED_630V18B2 = False
FD004_NORMALIZATION_EXECUTED_630V18B2 = False
FD004_WINDOW_BUILD_EXECUTED_630V18B2 = False
FD004_MC_EXECUTION_PERFORMED_630V18B2 = False
FD004_RELIABILITY_EXECUTED_630V18B2 = False
FD004_DEV_TARGETS_ERRORS_USED_630V18B2 = False
FD004_THRESHOLD_SELECTED_630V18B2 = False
FD004_POLICY_SELECTED_630V18B2 = False
FD004_TEST_CONTENTS_READ_630V18B2 = False
FD004_RUL_CONTENTS_READ_630V18B2 = False

all_checks_passed_630v18b2 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18B2 — FD004 EXACT DETERMINISTIC-"
    "INFERENCE SOURCE RECOVERY"
)

print("=" * 176)

print("Model forward performed           :", False)
print("Corruption executed               :", False)
print("Normalization executed            :", False)
print("Window build executed             :", False)
print("MC execution performed            :", False)
print("Reliability executed              :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- SOURCE RECOVERY ---")

print(
    "Raw source hits                   :",
    len(
        source_records_630v18b2
    ),
)

print(
    "Unique source texts               :",
    len(
        unique_records_630v18b2
    ),
)

print(
    "Smoke deterministic contexts      :",
    len(
        primary_contexts_630v18b2
    ),
)

print(
    "Frozen deterministic batch size   :",
    FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6[
        "deterministic_eval_batch_size"
    ],
)


print("\n--- UNIQUE SOURCE AUTHORITIES ---")

for idx_630v18b2, item_630v18b2 in enumerate(
    unique_records_630v18b2,
    start=1,
):

    print(
        f"\nSOURCE {idx_630v18b2}"
    )

    print(
        "  SHA-256                         :",
        item_630v18b2[
            "sha256"
        ],
    )

    print(
        "  Length                          :",
        item_630v18b2[
            "length"
        ],
    )

    print(
        "  Targets                         :",
        item_630v18b2[
            "targets"
        ],
    )

    print(
        "  Origins                         :",
        item_630v18b2[
            "origins"
        ][:10],
    )


print(
    "\n--- SMOKE DETERMINISTIC SOURCE CONTEXTS ---"
)

if primary_contexts_630v18b2:

    for idx_630v18b2, item_630v18b2 in enumerate(
        primary_contexts_630v18b2,
        start=1,
    ):

        print(
            f"\nSMOKE CONTEXT {idx_630v18b2}"
        )

        print(
            "Source SHA-256                   :",
            item_630v18b2[
                "source_sha256"
            ],
        )

        print(
            "Origins                          :",
            item_630v18b2[
                "origins"
            ][:10],
        )

        print(
            "Matched line                     :",
            item_630v18b2[
                "matched_line"
            ],
        )

        print(
            item_630v18b2[
                "context"
            ]
        )

else:

    print(
        "No exact source context containing "
        "FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2 was recovered."
    )


print(
    "\n--- OTHER DETERMINISTIC EVIDENCE CONTEXTS ---"
)

for target_630v18b2 in (
    "FD004_FIT_DETERMINISTIC_PREDICTION_630O",
    "FD004_CLEAN_DEV_PREDICTION_630K",
):

    matching_630v18b2 = [
        record_630v18b2
        for record_630v18b2
        in context_records_630v18b2
        if record_630v18b2[
            "target"
        ]
        ==
        target_630v18b2
    ]

    print(
        f"\n{target_630v18b2}"
    )

    print(
        "Contexts                         :",
        len(
            matching_630v18b2
        ),
    )

    # Print only the first context for each secondary target.
    if matching_630v18b2:

        print(
            matching_630v18b2[0][
                "context"
            ]
        )


print("\n--- AUDIT FINGERPRINT ---")

print(
    "Source-recovery audit SHA-256     :",
    FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2,
)

print(
    "\nPASS — prior deterministic-inference source evidence was "
    "located without executing the model."
)

print(
    "Next: freeze the deterministic 512-batch inference helper "
    "strictly from the recovered source before beginning bias execution."
)

In [ ]:
# STEP 6.30V-18C — FD004 DETERMINISTIC INFERENCE HELPER
# BIND + ONE FROZEN-SMOKE VALIDATION
#
# Exactly ONE deterministic model inference is performed,
# using the ALREADY-RESIDENT frozen smoke X.
#
# NO new corruption scenario.
# NO normalization/window rebuilding.
# NO MC.
# NO reliability execution.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import torch
import ast
import hashlib
import inspect
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18c = (
    "all_checks_passed_630v18b2",
    "FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2",
    "FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "FD004_SMOKE_X_DEV_630V2",
    "FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
    "FD004_TRAINED_MODEL_SHA256_630J",
)

missing_630v18c = [
    name_630v18c
    for name_630v18c in required_630v18c
    if name_630v18c not in globals()
]

if missing_630v18c:
    raise RuntimeError(
        "Missing STEP 6.30V-18C authority object(s): "
        + ", ".join(missing_630v18c)
    )

if not bool(all_checks_passed_630v18b2):
    raise RuntimeError(
        "STEP 6.30V-18B2 must remain passed."
    )


# =============================================================================
# 1) FROZEN CONTRACT GUARDS
# =============================================================================

execution_contract_630v18c = (
    FD004_FULL_CORRUPTION_EXECUTION_CONTRACT_630V6
)

det_batch_size_630v18c = int(
    execution_contract_630v18c[
        "deterministic_eval_batch_size"
    ]
)

if det_batch_size_630v18c != 512:
    raise RuntimeError(
        "Frozen deterministic batch size changed."
    )

if (
    str(
        execution_contract_630v18c[
            "model_authority"
        ]
    )
    !=
    "MODEL_FD004_630I"
):
    raise RuntimeError(
        "Frozen model authority changed."
    )

if (
    str(
        execution_contract_630v18c[
            "trained_model_sha256"
        ]
    )
    !=
    str(
        FD004_TRAINED_MODEL_SHA256_630J
    )
):
    raise RuntimeError(
        "Frozen trained-model authority changed."
    )


# =============================================================================
# 2) RECOVER THE EXACT HISTORICAL HELPER SOURCE FROM INPUT HISTORY
#    STATIC ONLY — this source is NOT exec'd.
# =============================================================================

HISTORICAL_HELPER_NAME_630V18C = (
    "_predict_deterministic_630v8"
)

historical_helper_hits_630v18c = []

if (
    "In" in globals()
    and
    isinstance(
        globals()["In"],
        list,
    )
):

    for (
        input_index_630v18c,
        input_source_630v18c,
    ) in enumerate(
        globals()["In"]
    ):

        if not isinstance(
            input_source_630v18c,
            str,
        ):
            continue

        if (
            HISTORICAL_HELPER_NAME_630V18C
            not in input_source_630v18c
        ):
            continue

        try:
            tree_630v18c = ast.parse(
                input_source_630v18c
            )
        except SyntaxError:
            continue

        for node_630v18c in ast.walk(
            tree_630v18c
        ):

            if (
                isinstance(
                    node_630v18c,
                    ast.FunctionDef,
                )
                and
                node_630v18c.name
                ==
                HISTORICAL_HELPER_NAME_630V18C
            ):

                lines_630v18c = (
                    input_source_630v18c.splitlines(
                        keepends=True
                    )
                )

                helper_source_630v18c = "".join(
                    lines_630v18c[
                        node_630v18c.lineno - 1:
                        node_630v18c.end_lineno
                    ]
                )

                historical_helper_hits_630v18c.append(
                    {
                        "input_index":
                            int(
                                input_index_630v18c
                            ),

                        "source":
                            helper_source_630v18c,

                        "sha256":
                            hashlib.sha256(
                                helper_source_630v18c.encode(
                                    "utf-8"
                                )
                            ).hexdigest(),
                    }
                )


# Deduplicate identical historical helper definitions.
historical_helper_by_sha_630v18c = {}

for item_630v18c in historical_helper_hits_630v18c:

    historical_helper_by_sha_630v18c.setdefault(
        item_630v18c[
            "sha256"
        ],
        item_630v18c,
    )

historical_helper_unique_630v18c = tuple(
    historical_helper_by_sha_630v18c.values()
)

if len(
    historical_helper_unique_630v18c
) != 1:
    raise RuntimeError(
        "Expected exactly one unique historical deterministic "
        "helper definition; found "
        f"{len(historical_helper_unique_630v18c)}."
    )

historical_helper_authority_630v18c = (
    historical_helper_unique_630v18c[0]
)

FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_630V18C = (
    historical_helper_authority_630v18c[
        "source"
    ]
)

FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_SHA256_630V18C = (
    historical_helper_authority_630v18c[
        "sha256"
    ]
)


# =============================================================================
# 3) BIND FD004 DETERMINISTIC HELPER
#
# Semantics copied strictly from the recovered historical helper:
# - np.asarray(..., float32)
# - model.eval()
# - torch.no_grad()
# - sequential batches
# - torch.from_numpy(...).to(device)
# - direct model forward
# - tuple/list length-one handling
# - flatten
# - concatenate
# - float32 return
# =============================================================================

def predict_fd004_deterministic_630v18c(
    X,
    model,
    device,
    batch_size=512,
):
    X = np.asarray(
        X,
        dtype=np.float32,
    )

    outputs = []

    model.eval()

    with torch.no_grad():

        for start in range(
            0,
            X.shape[0],
            batch_size,
        ):

            end = min(
                start + batch_size,
                X.shape[0],
            )

            xb = torch.from_numpy(
                X[
                    start:end
                ]
            ).to(
                device
            )

            yb = model(
                xb
            )

            if isinstance(
                yb,
                (
                    tuple,
                    list,
                ),
            ):

                if len(yb) != 1:
                    raise RuntimeError(
                        "Deterministic model returned a tuple/list "
                        "with unexpected length."
                    )

                yb = yb[0]

            yb = (
                yb
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            outputs.append(
                yb
            )

    return (
        np.concatenate(
            outputs,
            axis=0,
        )
        .astype(
            np.float32,
            copy=False,
        )
    )


# =============================================================================
# 4) STATIC BOUND-HELPER CHECKS
# =============================================================================

bound_signature_630v18c = str(
    inspect.signature(
        predict_fd004_deterministic_630v18c
    )
)

expected_signature_630v18c = (
    "(X, model, device, batch_size=512)"
)

if (
    bound_signature_630v18c
    !=
    expected_signature_630v18c
):
    raise RuntimeError(
        "Bound deterministic helper signature changed."
    )

if (
    predict_fd004_deterministic_630v18c.__defaults__
    !=
    (
        512,
    )
):
    raise RuntimeError(
        "Bound deterministic helper default batch size changed."
    )


# =============================================================================
# 5) MODEL STATE HASH HELPER
# =============================================================================

def _model_state_sha_630v18c(
    model_630v18c,
):

    h_630v18c = hashlib.sha256()

    for key_630v18c in sorted(
        model_630v18c.state_dict().keys()
    ):

        tensor_630v18c = (
            model_630v18c.state_dict()[
                key_630v18c
            ]
            .detach()
            .cpu()
            .contiguous()
        )

        arr_630v18c = (
            tensor_630v18c.numpy()
        )

        h_630v18c.update(
            key_630v18c.encode(
                "utf-8"
            )
        )

        h_630v18c.update(
            str(
                arr_630v18c.dtype
            ).encode(
                "utf-8"
            )
        )

        h_630v18c.update(
            str(
                tuple(
                    arr_630v18c.shape
                )
            ).encode(
                "utf-8"
            )
        )

        h_630v18c.update(
            arr_630v18c.tobytes(
                order="C"
            )
        )

    return h_630v18c.hexdigest()


# =============================================================================
# 6) FROZEN SMOKE INPUT AUTHORITY
# =============================================================================

smoke_X_630v18c = np.asarray(
    FD004_SMOKE_X_DEV_630V2,
    dtype=np.float32,
)

smoke_reference_prediction_630v18c = (
    np.asarray(
        FD004_SMOKE_DETERMINISTIC_PREDICTION_630V2,
        dtype=np.float32,
    )
    .reshape(-1)
)

if smoke_X_630v18c.shape != (
    10824,
    30,
    24,
):
    raise RuntimeError(
        "Frozen smoke X shape changed."
    )

if smoke_reference_prediction_630v18c.shape != (
    10824,
):
    raise RuntimeError(
        "Frozen smoke deterministic prediction shape changed."
    )

if not np.isfinite(
    smoke_X_630v18c
).all():
    raise RuntimeError(
        "Frozen smoke X contains non-finite values."
    )


# =============================================================================
# 7) EXACTLY ONE DETERMINISTIC VALIDATION INFERENCE
# =============================================================================

model_630v18c = (
    MODEL_FD004_630I
)

device_630v18c = (
    FD004_DEVICE_630I
)

model_mode_before_630v18c = bool(
    model_630v18c.training
)

if model_mode_before_630v18c:
    raise RuntimeError(
        "FD004 model must be in eval mode before deterministic "
        "helper validation."
    )

model_state_before_630v18c = (
    _model_state_sha_630v18c(
        model_630v18c
    )
)


# EXACTLY ONE model-inference call in this step.
smoke_prediction_rebuilt_630v18c = (
    predict_fd004_deterministic_630v18c(
        smoke_X_630v18c,
        model_630v18c,
        device_630v18c,
        batch_size=det_batch_size_630v18c,
    )
)


model_state_after_630v18c = (
    _model_state_sha_630v18c(
        model_630v18c
    )
)

model_mode_after_630v18c = bool(
    model_630v18c.training
)


# =============================================================================
# 8) EXACT REPRODUCTION CHECKS
# =============================================================================

prediction_shape_exact_630v18c = bool(
    smoke_prediction_rebuilt_630v18c.shape
    ==
    smoke_reference_prediction_630v18c.shape
)

prediction_values_exact_630v18c = bool(
    np.array_equal(
        smoke_prediction_rebuilt_630v18c,
        smoke_reference_prediction_630v18c,
    )
)

prediction_max_abs_diff_630v18c = float(
    np.max(
        np.abs(
            smoke_prediction_rebuilt_630v18c.astype(
                np.float64
            )
            -
            smoke_reference_prediction_630v18c.astype(
                np.float64
            )
        )
    )
)

prediction_finite_630v18c = bool(
    np.isfinite(
        smoke_prediction_rebuilt_630v18c
    ).all()
)

prediction_sha_630v18c = (
    hashlib.sha256(
        np.ascontiguousarray(
            smoke_prediction_rebuilt_630v18c,
            dtype=np.float32,
        ).tobytes(
            order="C"
        )
    ).hexdigest()
)

reference_prediction_sha_630v18c = (
    hashlib.sha256(
        np.ascontiguousarray(
            smoke_reference_prediction_630v18c,
            dtype=np.float32,
        ).tobytes(
            order="C"
        )
    ).hexdigest()
)

resident_smoke_sha_exact_630v18c = bool(
    reference_prediction_sha_630v18c
    ==
    str(
        FD004_SMOKE_PREDICTION_SHA256_630V2
    )
)

rebuilt_smoke_sha_exact_630v18c = bool(
    prediction_sha_630v18c
    ==
    str(
        FD004_SMOKE_PREDICTION_SHA256_630V2
    )
)

model_state_exact_630v18c = bool(
    model_state_before_630v18c
    ==
    model_state_after_630v18c
)

model_mode_exact_630v18c = bool(
    model_mode_before_630v18c
    ==
    model_mode_after_630v18c
)

validation_checks_630v18c = {
    "prediction_shape_exact":
        prediction_shape_exact_630v18c,

    "prediction_values_exact":
        prediction_values_exact_630v18c,

    "prediction_max_abs_diff_zero":
        prediction_max_abs_diff_630v18c
        ==
        0.0,

    "prediction_finite":
        prediction_finite_630v18c,

    "resident_smoke_sha_exact":
        resident_smoke_sha_exact_630v18c,

    "rebuilt_smoke_sha_exact":
        rebuilt_smoke_sha_exact_630v18c,

    "model_state_unchanged":
        model_state_exact_630v18c,

    "model_mode_unchanged":
        model_mode_exact_630v18c,
}

if not all(
    validation_checks_630v18c.values()
):
    failed_630v18c = [
        key_630v18c
        for key_630v18c, passed_630v18c
        in validation_checks_630v18c.items()
        if not passed_630v18c
    ]

    raise RuntimeError(
        "Deterministic helper validation failed: "
        + ", ".join(
            failed_630v18c
        )
    )


# =============================================================================
# 9) FREEZE DETERMINISTIC-INFERENCE CONTRACT
# =============================================================================

FD004_DETERMINISTIC_PREDICTOR_630V18C = (
    predict_fd004_deterministic_630v18c
)

FD004_DETERMINISTIC_INFERENCE_CONTRACT_630V18C = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18C",

    "historical_helper_name":
        HISTORICAL_HELPER_NAME_630V18C,

    "historical_helper_source_sha256":
        FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_SHA256_630V18C,

    "source_recovery_audit_sha256":
        str(
            FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2
        ),

    "model_authority":
        "MODEL_FD004_630I",

    "device_authority":
        "FD004_DEVICE_630I",

    "batch_size":
        512,

    "input_dtype":
        "float32",

    "model_mode_during_inference":
        "eval",

    "gradient_tracking":
        False,

    "output_dtype":
        "float32",

    "output_shape_rule":
        "(n_windows,)",

    "smoke_validation_inference_runs":
        1,

    "smoke_prediction_exact":
        prediction_values_exact_630v18c,

    "smoke_prediction_max_abs_diff":
        prediction_max_abs_diff_630v18c,

    "smoke_prediction_sha256":
        prediction_sha_630v18c,

    "model_state_unchanged":
        model_state_exact_630v18c,

    "model_mode_unchanged":
        model_mode_exact_630v18c,

    "new_corruption_scenarios_executed":
        0,

    "mc_execution_performed":
        False,

    "reliability_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18C = (
    hashlib.sha256(
        json.dumps(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_630V18C,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 10) FLAGS
# =============================================================================

FD004_DETERMINISTIC_HELPER_BOUND_630V18C = True

FD004_DETERMINISTIC_SMOKE_VALIDATION_RUNS_630V18C = 1

FD004_NEW_CORRUPTION_SCENARIOS_EXECUTED_630V18C = 0

FD004_MODEL_FORWARD_PERFORMED_630V18C = True
FD004_MC_EXECUTION_PERFORMED_630V18C = False
FD004_RELIABILITY_EXECUTED_630V18C = False

FD004_DEV_TARGETS_ERRORS_USED_630V18C = False
FD004_THRESHOLD_SELECTED_630V18C = False
FD004_POLICY_SELECTED_630V18C = False

FD004_TEST_CONTENTS_READ_630V18C = False
FD004_RUL_CONTENTS_READ_630V18C = False

all_checks_passed_630v18c = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18C — FD004 DETERMINISTIC INFERENCE HELPER "
    "BIND + FROZEN-SMOKE VALIDATION"
)

print("=" * 176)

print(
    "New corruption scenarios executed :",
    0,
)

print(
    "Deterministic validation runs      :",
    1,
)

print("MC execution performed            :", False)
print("Reliability executed              :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- HISTORICAL DETERMINISTIC AUTHORITY ---")

print(
    "Historical helper                 :",
    HISTORICAL_HELPER_NAME_630V18C,
)

print(
    "Historical source input index     :",
    historical_helper_authority_630v18c[
        "input_index"
    ],
)

print(
    "Historical helper source SHA-256  :",
    FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_SHA256_630V18C,
)

print(
    "Bound helper signature            :",
    bound_signature_630v18c,
)

print(
    "Frozen batch size                 :",
    det_batch_size_630v18c,
)


print("\n--- FROZEN SMOKE REPRODUCTION ---")

print(
    "Smoke X shape                     :",
    smoke_X_630v18c.shape,
)

print(
    "Prediction shape exact            :",
    prediction_shape_exact_630v18c,
)

print(
    "Prediction values exact           :",
    prediction_values_exact_630v18c,
)

print(
    "Prediction max abs diff           :",
    prediction_max_abs_diff_630v18c,
)

print(
    "Prediction finite                 :",
    prediction_finite_630v18c,
)

print(
    "Resident smoke SHA exact          :",
    resident_smoke_sha_exact_630v18c,
)

print(
    "Rebuilt smoke SHA exact           :",
    rebuilt_smoke_sha_exact_630v18c,
)

print(
    "Smoke prediction SHA-256          :",
    prediction_sha_630v18c,
)


print("\n--- MODEL SAFETY ---")

print(
    "Model state unchanged             :",
    model_state_exact_630v18c,
)

print(
    "Model mode unchanged              :",
    model_mode_exact_630v18c,
)

print(
    "Model mode after validation       :",
    (
        "train"
        if model_mode_after_630v18c
        else "eval"
    ),
)


print("\n--- CONTRACT ---")

print(
    "Deterministic helper bound        :",
    FD004_DETERMINISTIC_HELPER_BOUND_630V18C,
)

print(
    "Deterministic contract SHA-256    :",
    FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18C,
)

print(
    "\nPASS — the frozen deterministic 512-batch inference path "
    "has been bound and reproduced the existing FD004 smoke "
    "prediction exactly, without executing a new corruption scenario."
)

print(
    "Next: verify the live window-builder and reliability return "
    "interfaces, then execute only the 102 admissible bias scenarios."
)

In [ ]:
# STEP 6.30V-18C-R1 — FD004 DETERMINISTIC HELPER VALIDATION RECOVERY
#
# IMPORTANT:
# Reuses the prediction already computed by failed STEP 6.30V-18C.
#
# NO model forward pass.
# NO corruption.
# NO normalization/window execution.
# NO MC.
# NO reliability execution.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / RUL.

import numpy as np
import hashlib
import json
import inspect


# =============================================================================
# 0) REQUIRED RESIDENT OBJECTS FROM THE ALREADY-EXECUTED 18C
# =============================================================================

required_630v18cr1 = (
    "all_checks_passed_630v18b2",
    "predict_fd004_deterministic_630v18c",
    "smoke_prediction_rebuilt_630v18c",
    "smoke_reference_prediction_630v18c",
    "model_state_before_630v18c",
    "model_state_after_630v18c",
    "model_mode_before_630v18c",
    "model_mode_after_630v18c",
    "FD004_SMOKE_PREDICTION_SHA256_630V2",
    "FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_SHA256_630V18C",
    "FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2",
)

missing_630v18cr1 = [
    name_630v18cr1
    for name_630v18cr1 in required_630v18cr1
    if name_630v18cr1 not in globals()
]

if missing_630v18cr1:
    raise RuntimeError(
        "Cannot recover STEP 6.30V-18C without rerunning inference. "
        "Missing object(s): "
        + ", ".join(missing_630v18cr1)
    )

if not bool(all_checks_passed_630v18b2):
    raise RuntimeError(
        "STEP 6.30V-18B2 must remain passed."
    )


# =============================================================================
# 1) REUSE ALREADY-COMPUTED PREDICTIONS — NO MODEL CALL
# =============================================================================

rebuilt_630v18cr1 = (
    np.asarray(
        smoke_prediction_rebuilt_630v18c,
        dtype=np.float32,
    )
    .reshape(-1)
)

reference_630v18cr1 = (
    np.asarray(
        smoke_reference_prediction_630v18c,
        dtype=np.float32,
    )
    .reshape(-1)
)


shape_exact_630v18cr1 = bool(
    rebuilt_630v18cr1.shape
    ==
    reference_630v18cr1.shape
    ==
    (10824,)
)

values_exact_630v18cr1 = bool(
    np.array_equal(
        rebuilt_630v18cr1,
        reference_630v18cr1,
    )
)

max_abs_diff_630v18cr1 = float(
    np.max(
        np.abs(
            rebuilt_630v18cr1.astype(
                np.float64
            )
            -
            reference_630v18cr1.astype(
                np.float64
            )
        )
    )
)

finite_630v18cr1 = bool(
    np.isfinite(
        rebuilt_630v18cr1
    ).all()
    and
    np.isfinite(
        reference_630v18cr1
    ).all()
)


# =============================================================================
# 2) MODEL SAFETY FROM THE ALREADY-COMPLETED 18C FORWARD PASS
# =============================================================================

model_state_unchanged_630v18cr1 = bool(
    model_state_before_630v18c
    ==
    model_state_after_630v18c
)

model_mode_unchanged_630v18cr1 = bool(
    model_mode_before_630v18c
    ==
    model_mode_after_630v18c
)


# =============================================================================
# 3) DEFINE ONE EXPLICIT CANONICAL HASH CONVENTION FOR CURRENT EVIDENCE
#
# This is NOT assumed to be the convention used by the old
# FD004_SMOKE_PREDICTION_SHA256_630V2 object.
#
# It is used only to prove that resident and rebuilt equal arrays
# also produce the same hash under ONE explicitly defined convention.
# =============================================================================

def raw_float32_sha256_630v18cr1(
    arr_630v18cr1,
):
    arr_630v18cr1 = np.ascontiguousarray(
        np.asarray(
            arr_630v18cr1,
            dtype=np.float32,
        )
    )

    return hashlib.sha256(
        arr_630v18cr1.tobytes(
            order="C"
        )
    ).hexdigest()


reference_raw_sha_630v18cr1 = (
    raw_float32_sha256_630v18cr1(
        reference_630v18cr1
    )
)

rebuilt_raw_sha_630v18cr1 = (
    raw_float32_sha256_630v18cr1(
        rebuilt_630v18cr1
    )
)

canonical_hash_exact_630v18cr1 = bool(
    reference_raw_sha_630v18cr1
    ==
    rebuilt_raw_sha_630v18cr1
)


# =============================================================================
# 4) LEGACY SHA — DIAGNOSTIC ONLY, NEVER AN ADMISSION REQUIREMENT
# =============================================================================

legacy_stored_sha_630v18cr1 = str(
    FD004_SMOKE_PREDICTION_SHA256_630V2
)

legacy_hash_helper_present_630v18cr1 = bool(
    "sha256_array_630v2" in globals()
    and
    callable(
        globals().get(
            "sha256_array_630v2"
        )
    )
)

legacy_recomputed_sha_630v18cr1 = None
legacy_recomputed_matches_stored_630v18cr1 = None
legacy_hash_helper_source_630v18cr1 = None
legacy_hash_helper_source_sha_630v18cr1 = None
legacy_hash_diagnostic_error_630v18cr1 = None


if legacy_hash_helper_present_630v18cr1:

    legacy_helper_630v18cr1 = (
        globals()[
            "sha256_array_630v2"
        ]
    )

    try:

        # Hash computation only.
        # No model / corruption / inference.
        legacy_recomputed_sha_630v18cr1 = str(
            legacy_helper_630v18cr1(
                reference_630v18cr1
            )
        )

        legacy_recomputed_matches_stored_630v18cr1 = bool(
            legacy_recomputed_sha_630v18cr1
            ==
            legacy_stored_sha_630v18cr1
        )

    except Exception as exc_630v18cr1:

        legacy_hash_diagnostic_error_630v18cr1 = (
            repr(
                exc_630v18cr1
            )
        )

    try:

        legacy_hash_helper_source_630v18cr1 = (
            inspect.getsource(
                legacy_helper_630v18cr1
            )
        )

        legacy_hash_helper_source_sha_630v18cr1 = (
            hashlib.sha256(
                legacy_hash_helper_source_630v18cr1.encode(
                    "utf-8"
                )
            ).hexdigest()
        )

    except Exception:
        pass


# =============================================================================
# 5) TRUE VALIDATION REQUIREMENTS
#
# These are based on prediction equality/model safety.
# Legacy SHA convention is deliberately NOT included.
# =============================================================================

validation_checks_630v18cr1 = {

    "prediction_shape_exact":
        shape_exact_630v18cr1,

    "prediction_values_exact":
        values_exact_630v18cr1,

    "prediction_max_abs_diff_zero":
        max_abs_diff_630v18cr1
        ==
        0.0,

    "predictions_finite":
        finite_630v18cr1,

    "canonical_same_convention_hash_exact":
        canonical_hash_exact_630v18cr1,

    "model_state_unchanged":
        model_state_unchanged_630v18cr1,

    "model_mode_unchanged":
        model_mode_unchanged_630v18cr1,
}


if not all(
    validation_checks_630v18cr1.values()
):

    failed_630v18cr1 = [
        key_630v18cr1
        for (
            key_630v18cr1,
            passed_630v18cr1,
        )
        in validation_checks_630v18cr1.items()
        if not passed_630v18cr1
    ]

    raise RuntimeError(
        "TRUE deterministic-validation failure: "
        + ", ".join(
            failed_630v18cr1
        )
    )


# =============================================================================
# 6) FREEZE THE ALREADY-VERIFIED DETERMINISTIC HELPER
# =============================================================================

FD004_DETERMINISTIC_PREDICTOR_630V18CR1 = (
    predict_fd004_deterministic_630v18c
)


FD004_DETERMINISTIC_INFERENCE_CONTRACT_630V18CR1 = {

    "dataset":
        "FD004",

    "stage":
        "6.30V-18C-R1",

    "status":
        "DETERMINISTIC_HELPER_VALIDATED_BY_EXACT_FROZEN_SMOKE_REPRODUCTION",

    "historical_helper_source_sha256":
        str(
            FD004_HISTORICAL_DETERMINISTIC_HELPER_SOURCE_SHA256_630V18C
        ),

    "source_recovery_audit_sha256":
        str(
            FD004_DETERMINISTIC_SOURCE_RECOVERY_SHA256_630V18B2
        ),

    "model_authority":
        "MODEL_FD004_630I",

    "device_authority":
        "FD004_DEVICE_630I",

    "deterministic_batch_size":
        512,

    "input_dtype":
        "float32",

    "output_dtype":
        "float32",

    "validation_basis":
        (
            "exact array equality against already-resident "
            "frozen smoke prediction"
        ),

    "prediction_shape":
        tuple(
            rebuilt_630v18cr1.shape
        ),

    "prediction_values_exact":
        values_exact_630v18cr1,

    "prediction_max_abs_diff":
        max_abs_diff_630v18cr1,

    "canonical_raw_float32_sha256":
        rebuilt_raw_sha_630v18cr1,

    "legacy_stored_smoke_sha256":
        legacy_stored_sha_630v18cr1,

    "legacy_sha_convention_assumed":
        False,

    "legacy_hash_helper_present":
        legacy_hash_helper_present_630v18cr1,

    "legacy_recomputed_sha256":
        legacy_recomputed_sha_630v18cr1,

    "legacy_recomputed_matches_stored":
        legacy_recomputed_matches_stored_630v18cr1,

    "legacy_hash_helper_source_sha256":
        legacy_hash_helper_source_sha_630v18cr1,

    "legacy_hash_diagnostic_error":
        legacy_hash_diagnostic_error_630v18cr1,

    "model_state_unchanged":
        model_state_unchanged_630v18cr1,

    "model_mode_unchanged":
        model_mode_unchanged_630v18cr1,

    "additional_model_forward_runs":
        0,

    "new_corruption_scenarios_executed":
        0,

    "mc_execution_performed":
        False,

    "reliability_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1 = (
    hashlib.sha256(
        json.dumps(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_630V18CR1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode(
            "utf-8"
        )
    ).hexdigest()
)


# =============================================================================
# 7) FLAGS
# =============================================================================

FD004_DETERMINISTIC_HELPER_VALIDATED_630V18CR1 = True

FD004_ADDITIONAL_MODEL_FORWARD_RUNS_630V18CR1 = 0
FD004_NEW_CORRUPTION_SCENARIOS_EXECUTED_630V18CR1 = 0

FD004_MC_EXECUTION_PERFORMED_630V18CR1 = False
FD004_RELIABILITY_EXECUTED_630V18CR1 = False

FD004_DEV_TARGETS_ERRORS_USED_630V18CR1 = False
FD004_THRESHOLD_SELECTED_630V18CR1 = False
FD004_POLICY_SELECTED_630V18CR1 = False

FD004_TEST_CONTENTS_READ_630V18CR1 = False
FD004_RUL_CONTENTS_READ_630V18CR1 = False

all_checks_passed_630v18cr1 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18C-R1 — FD004 DETERMINISTIC "
    "HELPER VALIDATION RECOVERY"
)

print("=" * 176)

print(
    "Additional model forward runs     :",
    0,
)

print(
    "New corruption scenarios executed :",
    0,
)

print("MC execution performed            :", False)
print("Reliability executed              :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- TRUE DETERMINISTIC VALIDATION ---")

print(
    "Prediction shape exact            :",
    shape_exact_630v18cr1,
)

print(
    "Prediction values exact           :",
    values_exact_630v18cr1,
)

print(
    "Prediction max abs diff           :",
    max_abs_diff_630v18cr1,
)

print(
    "Predictions finite                :",
    finite_630v18cr1,
)

print(
    "Canonical same-convention SHA     :",
    canonical_hash_exact_630v18cr1,
)

print(
    "Canonical smoke SHA-256           :",
    rebuilt_raw_sha_630v18cr1,
)


print(
    "\n--- LEGACY SHA DIAGNOSTIC ONLY ---"
)

print(
    "Stored legacy smoke SHA           :",
    legacy_stored_sha_630v18cr1,
)

print(
    "Legacy hash helper resident       :",
    legacy_hash_helper_present_630v18cr1,
)

print(
    "Legacy recomputed SHA             :",
    legacy_recomputed_sha_630v18cr1,
)

print(
    "Legacy recomputed matches stored  :",
    legacy_recomputed_matches_stored_630v18cr1,
)

print(
    "Legacy helper source SHA          :",
    legacy_hash_helper_source_sha_630v18cr1,
)

print(
    "Legacy diagnostic error           :",
    legacy_hash_diagnostic_error_630v18cr1,
)


if (
    legacy_hash_helper_source_630v18cr1
    is not None
):

    print(
        "\n--- RESIDENT LEGACY HASH HELPER SOURCE ---"
    )

    print(
        legacy_hash_helper_source_630v18cr1
    )


print("\n--- MODEL SAFETY ---")

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18cr1,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18cr1,
)


print("\n--- CONTRACT ---")

print(
    "Deterministic helper validated    :",
    FD004_DETERMINISTIC_HELPER_VALIDATED_630V18CR1,
)

print(
    "Recovery contract SHA-256         :",
    FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1,
)


print(
    "\nPASS — deterministic inference is validated by exact "
    "equality to the already-resident frozen smoke prediction."
)

print(
    "The failed 18C assertion was a legacy-hash-convention "
    "assumption, not a prediction or model failure."
)

print(
    "Next: begin the frozen bias block using only the already-"
    "verified dispatcher, normalizer, window builder, deterministic "
    "predictor, MC helper, and reliability engine."
)

In [ ]:
# STEP 6.30V-18D0 — FD004 WINDOW / MC / RELIABILITY RETURN-CONTRACT AUDIT
#
# STATIC SOURCE INSPECTION ONLY.
# NO corruption / normalization / window construction.
# NO model inference / MC / reliability execution.
# NO DEV targets/errors.
# NO TEST / RUL.

import ast
import inspect
import textwrap
import hashlib
import json

required_630v18d0 = (
    "all_checks_passed_630v18cr1",
    "build_windows_fd004_630v2",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
)

missing_630v18d0 = [
    name for name in required_630v18d0
    if name not in globals()
]

if missing_630v18d0:
    raise RuntimeError(
        "Missing STEP 6.30V-18D0 authority object(s): "
        + ", ".join(missing_630v18d0)
    )

if not bool(all_checks_passed_630v18cr1):
    raise RuntimeError(
        "STEP 6.30V-18C-R1 must remain passed."
    )


def _recover_function_source_630v18d0(fn, expected_name):
    source = inspect.getsource(fn)
    source = textwrap.dedent(source)

    tree = ast.parse(source)

    matches = [
        node
        for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and node.name == expected_name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one source definition for {expected_name}; "
            f"found {len(matches)}."
        )

    return source, matches[0]


class _DirectReturnVisitor630V18D0(ast.NodeVisitor):
    def __init__(self):
        self.returns = []

    def visit_Return(self, node):
        self.returns.append(node)

    def visit_FunctionDef(self, node):
        return

    def visit_AsyncFunctionDef(self, node):
        return

    def visit_Lambda(self, node):
        return

    def visit_ClassDef(self, node):
        return


def _audit_function_630v18d0(fn, expected_name):
    source, fn_node = _recover_function_source_630v18d0(
        fn,
        expected_name,
    )

    visitor = _DirectReturnVisitor630V18D0()

    for stmt in fn_node.body:
        visitor.visit(stmt)

    return_records = []

    for return_node in visitor.returns:
        value = return_node.value

        if value is None:
            expression = None
            tuple_arity = 0
        else:
            expression = ast.get_source_segment(
                source,
                value,
            )
            tuple_arity = (
                len(value.elts)
                if isinstance(value, ast.Tuple)
                else 1
            )

        return_records.append(
            {
                "line":
                    int(return_node.lineno),

                "expression":
                    expression,

                "tuple_arity":
                    int(tuple_arity),
            }
        )

    return {
        "name":
            expected_name,

        "signature":
            str(
                inspect.signature(fn)
            ),

        "source":
            source,

        "source_sha256":
            hashlib.sha256(
                source.encode("utf-8")
            ).hexdigest(),

        "return_count":
            len(return_records),

        "returns":
            tuple(return_records),
    }


builder_audit_630v18d0 = _audit_function_630v18d0(
    build_windows_fd004_630v2,
    "build_windows_fd004_630v2",
)

mc_audit_630v18d0 = _audit_function_630v18d0(
    fd004_mc_dropout_predict_630v4r2,
    "fd004_mc_dropout_predict_630v4r2",
)

reliability_audit_630v18d0 = _audit_function_630v18d0(
    apply_fd004_final_reliability_chain_630v4r2,
    "apply_fd004_final_reliability_chain_630v4r2",
)


if builder_audit_630v18d0["return_count"] != 1:
    raise RuntimeError(
        "Window builder has an unexpected number of direct returns."
    )

if mc_audit_630v18d0["return_count"] != 1:
    raise RuntimeError(
        "MC helper has an unexpected number of direct returns."
    )

if reliability_audit_630v18d0["return_count"] != 1:
    raise RuntimeError(
        "Reliability helper has an unexpected number of direct returns."
    )


FD004_EXECUTION_HELPER_RETURN_AUDIT_630V18D0 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18D0",

    "window_builder": {
        "name":
            builder_audit_630v18d0["name"],

        "signature":
            builder_audit_630v18d0["signature"],

        "source_sha256":
            builder_audit_630v18d0["source_sha256"],

        "returns":
            builder_audit_630v18d0["returns"],
    },

    "mc_helper": {
        "name":
            mc_audit_630v18d0["name"],

        "signature":
            mc_audit_630v18d0["signature"],

        "source_sha256":
            mc_audit_630v18d0["source_sha256"],

        "returns":
            mc_audit_630v18d0["returns"],
    },

    "reliability_helper": {
        "name":
            reliability_audit_630v18d0["name"],

        "signature":
            reliability_audit_630v18d0["signature"],

        "source_sha256":
            reliability_audit_630v18d0["source_sha256"],

        "returns":
            reliability_audit_630v18d0["returns"],
    },

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helpers_executed":
        False,

    "corruption_executed":
        False,

    "normalization_executed":
        False,

    "window_construction_executed":
        False,

    "model_inference_performed":
        False,

    "mc_execution_performed":
        False,

    "reliability_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0 = (
    hashlib.sha256(
        json.dumps(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_630V18D0,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_HELPERS_EXECUTED_630V18D0 = False
FD004_CORRUPTION_EXECUTED_630V18D0 = False
FD004_MODEL_INFERENCE_PERFORMED_630V18D0 = False
FD004_MC_EXECUTION_PERFORMED_630V18D0 = False
FD004_RELIABILITY_EXECUTED_630V18D0 = False
FD004_DEV_TARGETS_ERRORS_USED_630V18D0 = False
FD004_TEST_CONTENTS_READ_630V18D0 = False
FD004_RUL_CONTENTS_READ_630V18D0 = False

all_checks_passed_630v18d0 = True


print("=" * 176)

print(
    "STEP 6.30V-18D0 — FD004 WINDOW / MC / "
    "RELIABILITY RETURN-CONTRACT AUDIT"
)

print("=" * 176)

print("Helpers executed                  :", False)
print("Corruption executed               :", False)
print("Normalization executed            :", False)
print("Window construction executed      :", False)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("Reliability executed              :", False)
print("DEV targets/errors used           :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


for label, audit in (
    ("WINDOW BUILDER", builder_audit_630v18d0),
    ("MC HELPER", mc_audit_630v18d0),
    ("RELIABILITY HELPER", reliability_audit_630v18d0),
):
    print(
        f"\n--- {label} ---"
    )

    print(
        "Name                              :",
        audit["name"],
    )

    print(
        "Signature                         :",
        audit["signature"],
    )

    print(
        "Source SHA-256                    :",
        audit["source_sha256"],
    )

    print(
        "Direct return count               :",
        audit["return_count"],
    )

    for idx, record in enumerate(
        audit["returns"],
        start=1,
    ):
        print(
            f"Return {idx} line / tuple arity     :",
            (
                record["line"],
                record["tuple_arity"],
            ),
        )

        print(
            f"Return {idx} expression             :",
            record["expression"],
        )


print(
    "\n--- WINDOW BUILDER EXACT SOURCE ---"
)

print(
    builder_audit_630v18d0[
        "source"
    ]
)


print(
    "\n--- RELIABILITY HELPER EXACT SOURCE ---"
)

print(
    reliability_audit_630v18d0[
        "source"
    ]
)


print("\n--- AUDIT FINGERPRINT ---")

print(
    "Return-contract audit SHA-256     :",
    FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0,
)

print(
    "\nPASS — execution-helper return contracts were recovered "
    "statically. Nothing was executed."
)

print(
    "Next: construct the first bias-family execution cell using "
    "only these exact return contracts."
)

In [ ]:
# STEP 6.30V-18E — FD004 BIAS FAMILY EXECUTION
# 102 admissible bias scenarios only.
#
# LABEL-FREE EXECUTION:
# - executes corruption -> normalization -> windows -> deterministic prediction
#   -> 50-pass MC dropout -> reliability chain
# - NO DEV targets/errors
# - NO threshold/policy selection
# - NO TEST / official RUL access
#
# Retain per frozen contract:
# - deterministic prediction
# - MC STD
# - final selective risk
# Hash full MC matrix, then discard it.

import numpy as np
import pandas as pd
import torch
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18e = (
    "all_checks_passed_630v18d0",
    "FD004_381_EXECUTION_ADMITTED_630V18A",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_DETERMINISTIC_PREDICTOR_630V18CR1",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
    "FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0",
)

missing_630v18e = [
    name_630v18e
    for name_630v18e in required_630v18e
    if name_630v18e not in globals()
]

if missing_630v18e:
    raise RuntimeError(
        "Missing STEP 6.30V-18E authority object(s): "
        + ", ".join(missing_630v18e)
    )

if not bool(all_checks_passed_630v18d0):
    raise RuntimeError(
        "STEP 6.30V-18D0 must remain passed."
    )

if not bool(FD004_381_EXECUTION_ADMITTED_630V18A):
    raise RuntimeError(
        "381-scenario execution was not admitted."
    )


# =============================================================================
# 1) EXACT PRE-BLOCK AUTHORITY GUARDS
# =============================================================================

EXPECTED_ADMISSION_SHA256_630V18E = (
    "41bb57b7bbaf90e80c96b9e75eeeac81bae2b8df7a44aea1d09df84bee3f56d4"
)

EXPECTED_RETURN_AUDIT_SHA256_630V18E = (
    "3c60101b5c7d36c40bebbb3b4542255a0aef8125970a673de480641b43c1e1fe"
)

if (
    str(FD004_381_EXECUTION_ADMISSION_SHA256_630V18A)
    != EXPECTED_ADMISSION_SHA256_630V18E
):
    raise RuntimeError(
        "FD004 execution-admission fingerprint changed."
    )

if (
    str(FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0)
    != EXPECTED_RETURN_AUDIT_SHA256_630V18E
):
    raise RuntimeError(
        "FD004 helper return-contract fingerprint changed."
    )


# =============================================================================
# 2) FREEZE BIAS BLOCK FROM EXECUTION MANIFEST
# =============================================================================

execution_manifest_630v18e = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

bias_manifest_630v18e = (
    execution_manifest_630v18e.loc[
        execution_manifest_630v18e[
            "family"
        ].astype(str)
        ==
        "bias"
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(bias_manifest_630v18e) != 102:
    raise RuntimeError(
        "Expected exactly 102 admissible bias scenarios."
    )

bias_numbers_630v18e = (
    bias_manifest_630v18e[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

if (
    int(bias_numbers_630v18e.min()) != 7
    or
    int(bias_numbers_630v18e.max()) != 126
):
    raise RuntimeError(
        "Frozen bias scenario-number range changed."
    )

if not np.all(
    bias_manifest_630v18e[
        "family"
    ].astype(str).to_numpy()
    ==
    "bias"
):
    raise RuntimeError(
        "Non-bias row entered the bias execution block."
    )

if not (
    bias_manifest_630v18e[
        "alpha"
    ].notna().all()
    and
    bias_manifest_630v18e[
        "direction"
    ].notna().all()
):
    raise RuntimeError(
        "Bias manifest has missing alpha/direction values."
    )


# =============================================================================
# 3) IMMUTABLE EXECUTION AUTHORITIES / PRE-BLOCK STATE
# =============================================================================

base_630v18e = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v18e = (
    base_630v18e.copy(deep=True)
)

regime_ids_630v18e = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

expected_window_units_630v18e = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

expected_endpoint_cycles_630v18e = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if expected_window_units_630v18e.shape != (
    10824,
):
    raise RuntimeError(
        "Frozen DEV window-unit authority shape changed."
    )

if expected_endpoint_cycles_630v18e.shape != (
    10824,
):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle authority shape changed."
    )

model_630v18e = (
    MODEL_FD004_630I
)

device_630v18e = (
    FD004_DEVICE_630I
)

if bool(model_630v18e.training):
    raise RuntimeError(
        "FD004 model must be in eval mode before bias execution."
    )


def _model_state_sha_630v18e(model_value_630v18e):
    h_630v18e = hashlib.sha256()

    state_630v18e = (
        model_value_630v18e.state_dict()
    )

    for key_630v18e in sorted(
        state_630v18e.keys()
    ):
        tensor_630v18e = (
            state_630v18e[
                key_630v18e
            ]
            .detach()
            .cpu()
            .contiguous()
        )

        arr_630v18e = tensor_630v18e.numpy()

        h_630v18e.update(
            key_630v18e.encode("utf-8")
        )

        h_630v18e.update(
            str(
                arr_630v18e.dtype
            ).encode("utf-8")
        )

        h_630v18e.update(
            str(
                tuple(
                    arr_630v18e.shape
                )
            ).encode("utf-8")
        )

        h_630v18e.update(
            arr_630v18e.tobytes(
                order="C"
            )
        )

    return h_630v18e.hexdigest()


model_state_before_630v18e = (
    _model_state_sha_630v18e(
        model_630v18e
    )
)

model_mode_before_630v18e = bool(
    model_630v18e.training
)


# =============================================================================
# 4) PREALLOCATE ONLY THE FROZEN RETAINED PER-SCENARIO ARRAYS
# =============================================================================

n_scenarios_630v18e = int(
    len(bias_manifest_630v18e)
)

n_windows_630v18e = 10824

bias_pred_matrix_630v18e = np.empty(
    (
        n_scenarios_630v18e,
        n_windows_630v18e,
    ),
    dtype=np.float32,
)

bias_mc_std_matrix_630v18e = np.empty(
    (
        n_scenarios_630v18e,
        n_windows_630v18e,
    ),
    dtype=np.float64,
)

bias_final_risk_matrix_630v18e = np.empty(
    (
        n_scenarios_630v18e,
        n_windows_630v18e,
    ),
    dtype=np.float64,
)

bias_mc_matrix_hashes_630v18e = []
bias_scenario_evidence_630v18e = []


# =============================================================================
# 5) EXECUTE EXACTLY THE 102 ADMISSIBLE BIAS SCENARIOS
# =============================================================================

print("=" * 176)
print(
    "STEP 6.30V-18E — FD004 BIAS FAMILY EXECUTION "
    "(102 ADMISSIBLE SCENARIOS)"
)
print("=" * 176)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)
print(
    "\nStarting frozen bias block         :",
    n_scenarios_630v18e,
    "scenarios",
)

for row_index_630v18e, row_630v18e in (
    bias_manifest_630v18e.iterrows()
):

    scenario_id_630v18e = str(
        row_630v18e[
            "scenario_id"
        ]
    )

    scenario_number_630v18e = int(
        row_630v18e[
            "scenario_number"
        ]
    )

    sensor_630v18e = str(
        row_630v18e[
            "sensor"
        ]
    )

    alpha_630v18e = float(
        row_630v18e[
            "alpha"
        ]
    )

    direction_630v18e = int(
        row_630v18e[
            "direction"
        ]
    )

    # -------------------------------------------------------------------------
    # A) Frozen corruption dispatcher
    # -------------------------------------------------------------------------

    corrupted_raw_630v18e = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v18e,
            row_630v18e,
        )
    )

    if not isinstance(
        corrupted_raw_630v18e,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: dispatcher did not return DataFrame."
        )

    if corrupted_raw_630v18e.shape != (
        12274,
        27,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: corrupted raw shape changed."
        )

    if not np.array_equal(
        corrupted_raw_630v18e[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_ids_630v18e,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: regime_id changed under bias."
        )

    # -------------------------------------------------------------------------
    # B) Frozen normalization bridge
    # -------------------------------------------------------------------------

    normalized_630v18e = (
        normalize_fd004_raw_630v2(
            corrupted_raw_630v18e,
            regime_ids_630v18e,
        )
    )

    # -------------------------------------------------------------------------
    # C) Frozen window builder
    # -------------------------------------------------------------------------

    (
        X_630v18e,
        unit_ids_630v18e,
        endpoint_cycles_630v18e,
    ) = build_windows_fd004_630v2(
        normalized_630v18e
    )

    X_630v18e = np.asarray(
        X_630v18e,
        dtype=np.float32,
    )

    if X_630v18e.shape != (
        10824,
        30,
        24,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: window tensor shape changed."
        )

    if not np.isfinite(
        X_630v18e
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18e}: non-finite model input."
        )

    if not np.array_equal(
        np.asarray(
            unit_ids_630v18e,
            dtype=np.int64,
        ).reshape(-1),
        expected_window_units_630v18e,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: window-unit membership changed."
        )

    if not np.array_equal(
        np.asarray(
            endpoint_cycles_630v18e,
            dtype=np.int64,
        ).reshape(-1),
        expected_endpoint_cycles_630v18e,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: endpoint-cycle alignment changed."
        )

    # -------------------------------------------------------------------------
    # D) Frozen deterministic inference
    # -------------------------------------------------------------------------

    deterministic_pred_630v18e = np.asarray(
        FD004_DETERMINISTIC_PREDICTOR_630V18CR1(
            X_630v18e,
            model_630v18e,
            device_630v18e,
            batch_size=512,
        ),
        dtype=np.float32,
    ).reshape(-1)

    if deterministic_pred_630v18e.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: deterministic prediction shape changed."
        )

    if not np.isfinite(
        deterministic_pred_630v18e
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18e}: non-finite deterministic prediction."
        )

    # -------------------------------------------------------------------------
    # E) Frozen 50-pass MC dropout
    # -------------------------------------------------------------------------

    (
        mc_matrix_630v18e,
        mc_mean_630v18e,
        mc_std_630v18e,
    ) = fd004_mc_dropout_predict_630v4r2(
        X_630v18e,
        model_630v18e,
        device_630v18e,
    )

    mc_matrix_630v18e = np.asarray(
        mc_matrix_630v18e,
        dtype=np.float32,
    )

    mc_std_630v18e = np.asarray(
        mc_std_630v18e,
        dtype=np.float64,
    ).reshape(-1)

    if mc_matrix_630v18e.shape != (
        50,
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: MC matrix shape changed."
        )

    if mc_std_630v18e.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: MC STD shape changed."
        )

    if not (
        np.isfinite(
            mc_matrix_630v18e
        ).all()
        and
        np.isfinite(
            mc_std_630v18e
        ).all()
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: non-finite MC output."
        )

    if np.any(
        mc_std_630v18e < 0.0
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: negative MC STD."
        )

    mc_matrix_sha_630v18e = str(
        sha256_array_630v2(
            mc_matrix_630v18e
        )
    )

    # -------------------------------------------------------------------------
    # F) Frozen label-free reliability chain
    # -------------------------------------------------------------------------

    (
        phase_ids_630v18e,
        adjusted_630v18e,
        g1_630v18e,
        final_risk_630v18e,
    ) = apply_fd004_final_reliability_chain_630v4r2(
        deterministic_pred_630v18e,
        mc_std_630v18e,
    )

    final_risk_630v18e = np.asarray(
        final_risk_630v18e,
        dtype=np.float64,
    ).reshape(-1)

    if final_risk_630v18e.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: final risk shape changed."
        )

    if not np.isfinite(
        final_risk_630v18e
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18e}: non-finite final risk."
        )

    if np.any(
        (final_risk_630v18e < 0.0)
        |
        (final_risk_630v18e > 1.0)
    ):
        raise RuntimeError(
            f"{scenario_id_630v18e}: final risk escaped [0,1]."
        )

    # -------------------------------------------------------------------------
    # G) Retain only frozen required arrays + evidence hashes
    # -------------------------------------------------------------------------

    bias_pred_matrix_630v18e[
        row_index_630v18e
    ] = deterministic_pred_630v18e

    bias_mc_std_matrix_630v18e[
        row_index_630v18e
    ] = mc_std_630v18e

    bias_final_risk_matrix_630v18e[
        row_index_630v18e
    ] = final_risk_630v18e

    bias_mc_matrix_hashes_630v18e.append(
        mc_matrix_sha_630v18e
    )

    bias_scenario_evidence_630v18e.append(
        {
            "scenario_id":
                scenario_id_630v18e,

            "scenario_number":
                scenario_number_630v18e,

            "family":
                "bias",

            "sensor":
                sensor_630v18e,

            "alpha":
                alpha_630v18e,

            "direction":
                direction_630v18e,

            "deterministic_prediction_sha256":
                str(
                    sha256_array_630v2(
                        deterministic_pred_630v18e
                    )
                ),

            "mc_matrix_sha256":
                mc_matrix_sha_630v18e,

            "mc_std_sha256":
                str(
                    sha256_array_630v2(
                        mc_std_630v18e
                    )
                ),

            "final_selective_risk_sha256":
                str(
                    sha256_array_630v2(
                        final_risk_630v18e
                    )
                ),
        }
    )

    # Full MC matrix MUST NOT be retained.
    del mc_matrix_630v18e
    del mc_mean_630v18e

    # Intermediate reliability products are not retained by the frozen contract.
    del phase_ids_630v18e
    del adjusted_630v18e
    del g1_630v18e

    if (
        (row_index_630v18e + 1) == 1
        or
        (row_index_630v18e + 1) % 10 == 0
        or
        (row_index_630v18e + 1) == n_scenarios_630v18e
    ):
        print(
            f"Completed bias scenario "
            f"{row_index_630v18e + 1:3d}/"
            f"{n_scenarios_630v18e}: "
            f"{scenario_id_630v18e}"
        )


# =============================================================================
# 6) POST-BLOCK SCIENTIFIC SAFETY CHECKS
# =============================================================================

if not np.isfinite(
    bias_pred_matrix_630v18e
).all():
    raise RuntimeError(
        "Retained bias deterministic predictions contain non-finite values."
    )

if not np.isfinite(
    bias_mc_std_matrix_630v18e
).all():
    raise RuntimeError(
        "Retained bias MC STD contains non-finite values."
    )

if not np.isfinite(
    bias_final_risk_matrix_630v18e
).all():
    raise RuntimeError(
        "Retained bias final risk contains non-finite values."
    )

if np.any(
    bias_mc_std_matrix_630v18e < 0.0
):
    raise RuntimeError(
        "Retained bias MC STD contains negative values."
    )

if np.any(
    (bias_final_risk_matrix_630v18e < 0.0)
    |
    (bias_final_risk_matrix_630v18e > 1.0)
):
    raise RuntimeError(
        "Retained bias final risk escaped [0,1]."
    )

if len(
    bias_mc_matrix_hashes_630v18e
) != 102:
    raise RuntimeError(
        "Bias MC matrix hash count changed."
    )

if len(
    bias_scenario_evidence_630v18e
) != 102:
    raise RuntimeError(
        "Bias scenario evidence count changed."
    )

base_unchanged_630v18e = bool(
    base_630v18e.equals(
        base_snapshot_630v18e
    )
)

if not base_unchanged_630v18e:
    raise RuntimeError(
        "Frozen corruption base changed during bias execution."
    )

model_state_after_630v18e = (
    _model_state_sha_630v18e(
        model_630v18e
    )
)

model_mode_after_630v18e = bool(
    model_630v18e.training
)

model_state_unchanged_630v18e = bool(
    model_state_before_630v18e
    ==
    model_state_after_630v18e
)

model_mode_unchanged_630v18e = bool(
    model_mode_before_630v18e
    ==
    model_mode_after_630v18e
)

if not model_state_unchanged_630v18e:
    raise RuntimeError(
        "Model state changed during bias block."
    )

if not model_mode_unchanged_630v18e:
    raise RuntimeError(
        "Model mode changed during bias block."
    )


# =============================================================================
# 7) FREEZE COMPLETED BIAS-BLOCK EVIDENCE
# =============================================================================

FD004_BIAS_SCENARIO_MANIFEST_630V18E = (
    bias_manifest_630v18e.copy()
)

FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E = (
    bias_pred_matrix_630v18e
)

FD004_BIAS_MC_STD_630V18E = (
    bias_mc_std_matrix_630v18e
)

FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E = (
    bias_final_risk_matrix_630v18e
)

FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E = tuple(
    bias_mc_matrix_hashes_630v18e
)

FD004_BIAS_SCENARIO_EVIDENCE_630V18E = tuple(
    bias_scenario_evidence_630v18e
)


FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E = str(
    sha256_array_630v2(
        FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E
    )
)

FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E = str(
    sha256_array_630v2(
        FD004_BIAS_MC_STD_630V18E
    )
)

FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E = str(
    sha256_array_630v2(
        FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E
    )
)


bias_block_contract_630v18e = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18E",

    "family":
        "bias",

    "status":
        "BIAS_BLOCK_COMPLETE_LABEL_FREE",

    "scenario_count":
        102,

    "scenario_number_min":
        int(
            bias_numbers_630v18e.min()
        ),

    "scenario_number_max":
        int(
            bias_numbers_630v18e.max()
        ),

    "deterministic_prediction_shape":
        tuple(
            FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E.shape
        ),

    "mc_std_shape":
        tuple(
            FD004_BIAS_MC_STD_630V18E.shape
        ),

    "final_selective_risk_shape":
        tuple(
            FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E.shape
        ),

    "full_mc_matrix_hash_count":
        len(
            FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E
        ),

    "full_mc_matrices_retained":
        False,

    "prediction_block_sha256":
        FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E,

    "mc_std_block_sha256":
        FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E,

    "final_risk_block_sha256":
        FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E,

    "admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helper_return_audit_sha256":
        str(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0
        ),

    "corruption_base_unchanged":
        base_unchanged_630v18e,

    "model_state_unchanged":
        model_state_unchanged_630v18e,

    "model_mode_unchanged":
        model_mode_unchanged_630v18e,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_BIAS_BLOCK_CONTRACT_630V18E = (
    bias_block_contract_630v18e
)

FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E = (
    hashlib.sha256(
        json.dumps(
            bias_block_contract_630v18e,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 8) PERSISTENCE CHECKPOINT — NON-SCIENTIFIC
#
# Failure to persist does NOT invalidate the completed scientific block.
# =============================================================================

checkpoint_saved_630v18e = False
checkpoint_error_630v18e = None
checkpoint_npz_path_630v18e = None
checkpoint_json_path_630v18e = None
checkpoint_npz_sha256_630v18e = None
checkpoint_json_sha256_630v18e = None

try:
    checkpoint_dir_630v18e = Path(
        r"C:\RUL_Reliability_Research\checkpoints"
    )

    checkpoint_dir_630v18e.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_npz_path_630v18e = (
        checkpoint_dir_630v18e
        /
        "fd004_bias_block_630v18e_v1.npz"
    )

    checkpoint_json_path_630v18e = (
        checkpoint_dir_630v18e
        /
        "fd004_bias_block_630v18e_v1.json"
    )

    np.savez_compressed(
        checkpoint_npz_path_630v18e,
        scenario_number=bias_manifest_630v18e[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
        scenario_id=bias_manifest_630v18e[
            "scenario_id"
        ].astype(str).to_numpy(),
        deterministic_prediction=FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E,
        mc_std=FD004_BIAS_MC_STD_630V18E,
        final_selective_risk=FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E,
        full_mc_matrix_sha256=np.asarray(
            FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E,
            dtype="<U64",
        ),
    )

    checkpoint_metadata_630v18e = {
        "dataset":
            "FD004",

        "stage":
            "6.30V-18E",

        "family":
            "bias",

        "scenario_count":
            102,

        "block_contract_sha256":
            FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E,

        "prediction_block_sha256":
            FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E,

        "mc_std_block_sha256":
            FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E,

        "final_risk_block_sha256":
            FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E,

        "scenario_evidence":
            list(
                FD004_BIAS_SCENARIO_EVIDENCE_630V18E
            ),

        "dev_targets_errors_used":
            False,

        "threshold_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    }

    checkpoint_json_path_630v18e.write_text(
        json.dumps(
            checkpoint_metadata_630v18e,
            sort_keys=True,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    def _file_sha256_630v18e(path_630v18e):
        h_630v18e = hashlib.sha256()

        with open(
            path_630v18e,
            "rb",
        ) as file_630v18e:
            while True:
                chunk_630v18e = file_630v18e.read(
                    1024 * 1024
                )

                if not chunk_630v18e:
                    break

                h_630v18e.update(
                    chunk_630v18e
                )

        return h_630v18e.hexdigest()

    checkpoint_npz_sha256_630v18e = (
        _file_sha256_630v18e(
            checkpoint_npz_path_630v18e
        )
    )

    checkpoint_json_sha256_630v18e = (
        _file_sha256_630v18e(
            checkpoint_json_path_630v18e
        )
    )

    checkpoint_saved_630v18e = True

except Exception as exc_630v18e:
    checkpoint_error_630v18e = repr(
        exc_630v18e
    )


FD004_BIAS_CHECKPOINT_SAVED_630V18E = bool(
    checkpoint_saved_630v18e
)

FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E = (
    None
    if checkpoint_npz_path_630v18e is None
    else str(
        checkpoint_npz_path_630v18e
    )
)

FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E = (
    None
    if checkpoint_json_path_630v18e is None
    else str(
        checkpoint_json_path_630v18e
    )
)

FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E = (
    checkpoint_npz_sha256_630v18e
)

FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E = (
    checkpoint_json_sha256_630v18e
)

FD004_BIAS_CHECKPOINT_ERROR_630V18E = (
    checkpoint_error_630v18e
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD004_BIAS_BLOCK_COMPLETE_630V18E = True
FD004_BIAS_SCENARIOS_EXECUTED_630V18E = 102

FD004_DEV_TARGETS_ERRORS_USED_630V18E = False
FD004_THRESHOLD_SELECTED_630V18E = False
FD004_POLICY_SELECTED_630V18E = False
FD004_TEST_CONTENTS_READ_630V18E = False
FD004_RUL_CONTENTS_READ_630V18E = False

all_checks_passed_630v18e = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("\n--- BIAS BLOCK COMPLETE ---")

print(
    "Bias scenarios executed           :",
    FD004_BIAS_SCENARIOS_EXECUTED_630V18E,
)

print(
    "Deterministic retained shape      :",
    FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E.shape,
)

print(
    "MC STD retained shape             :",
    FD004_BIAS_MC_STD_630V18E.shape,
)

print(
    "Final risk retained shape         :",
    FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E.shape,
)

print(
    "Full MC matrix hashes retained    :",
    len(
        FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E
    ),
)

print(
    "Full MC matrices retained         :",
    False,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18e,
)

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18e,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18e,
)

print(
    "Prediction block SHA-256          :",
    FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E,
)

print(
    "MC STD block SHA-256              :",
    FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E,
)

print(
    "Final risk block SHA-256          :",
    FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E,
)

print(
    "Bias block contract SHA-256       :",
    FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E,
)


print("\n--- LABEL-FREE BLOCK SUMMARY ---")

print(
    "MC STD min / median / mean / max  :",
    (
        float(
            np.min(
                FD004_BIAS_MC_STD_630V18E
            )
        ),
        float(
            np.median(
                FD004_BIAS_MC_STD_630V18E
            )
        ),
        float(
            np.mean(
                FD004_BIAS_MC_STD_630V18E
            )
        ),
        float(
            np.max(
                FD004_BIAS_MC_STD_630V18E
            )
        ),
    ),
)

print(
    "Risk min / median / mean / max    :",
    (
        float(
            np.min(
                FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E
            )
        ),
        float(
            np.median(
                FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E
            )
        ),
        float(
            np.mean(
                FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E
            )
        ),
        float(
            np.max(
                FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E
            )
        ),
    ),
)


print("\n--- CHECKPOINT ---")

print(
    "Checkpoint saved                  :",
    FD004_BIAS_CHECKPOINT_SAVED_630V18E,
)

print(
    "NPZ path                          :",
    FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E,
)

print(
    "JSON path                         :",
    FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E,
)

print(
    "NPZ SHA-256                       :",
    FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E,
)

print(
    "JSON SHA-256                      :",
    FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E,
)

print(
    "Checkpoint error                  :",
    FD004_BIAS_CHECKPOINT_ERROR_630V18E,
)


print("\n--- LEAKAGE / POLICY GUARDS ---")

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nPASS — the 102 admissible FD004 bias scenarios are complete "
    "under the frozen label-free execution contract."
)

if checkpoint_saved_630v18e:
    print(
        "The completed bias block is also checkpointed. "
        "Next: audit this block evidence, then proceed to noise."
    )
else:
    print(
        "Scientific bias-block execution is complete, but persistence "
        "checkpointing failed. Do NOT rerun the bias block; repair only "
        "the checkpoint before proceeding."
    )

In [ ]:
# STEP 6.30V-18F — FD004 BIAS BLOCK EVIDENCE + CHECKPOINT ROUND-TRIP AUDIT
#
# ZERO SCIENTIFIC EXECUTION:
# - no corruption
# - no normalization/windows
# - no deterministic model inference
# - no MC
# - no reliability
# - no DEV targets/errors
# - no threshold/policy
# - no TEST / official RUL
#
# This cell audits only the already-completed STEP 6.30V-18E bias evidence
# and its persistence checkpoint.

import numpy as np
import pandas as pd
import hashlib
import json
from pathlib import Path

# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18f = (
    "all_checks_passed_630v18e",
    "FD004_BIAS_BLOCK_COMPLETE_630V18E",
    "FD004_BIAS_SCENARIOS_EXECUTED_630V18E",
    "FD004_BIAS_SCENARIO_MANIFEST_630V18E",
    "FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E",
    "FD004_BIAS_MC_STD_630V18E",
    "FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E",
    "FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E",
    "FD004_BIAS_SCENARIO_EVIDENCE_630V18E",
    "FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E",
    "FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E",
    "FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E",
    "FD004_BIAS_BLOCK_CONTRACT_630V18E",
    "FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E",
    "FD004_BIAS_CHECKPOINT_SAVED_630V18E",
    "FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E",
    "FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E",
    "FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E",
    "FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "sha256_array_630v2",
)

missing_630v18f = [
    name for name in required_630v18f
    if name not in globals()
]

if missing_630v18f:
    raise RuntimeError(
        "Missing STEP 6.30V-18F authority object(s): "
        + ", ".join(missing_630v18f)
    )

if not bool(all_checks_passed_630v18e):
    raise RuntimeError("STEP 6.30V-18E must remain passed.")

if not bool(FD004_BIAS_BLOCK_COMPLETE_630V18E):
    raise RuntimeError("Bias block is not marked complete.")

if int(FD004_BIAS_SCENARIOS_EXECUTED_630V18E) != 102:
    raise RuntimeError("Bias executed-scenario count changed.")

if not bool(FD004_BIAS_CHECKPOINT_SAVED_630V18E):
    raise RuntimeError("Bias checkpoint was not successfully saved.")


# =============================================================================
# 1) RESIDENT EVIDENCE SHAPES / FINITENESS
# =============================================================================

pred_630v18f = np.asarray(
    FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E,
    dtype=np.float32,
)

std_630v18f = np.asarray(
    FD004_BIAS_MC_STD_630V18E,
    dtype=np.float64,
)

risk_630v18f = np.asarray(
    FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E,
    dtype=np.float64,
)

expected_shape_630v18f = (102, 10824)

shape_checks_630v18f = {
    "prediction": pred_630v18f.shape == expected_shape_630v18f,
    "mc_std": std_630v18f.shape == expected_shape_630v18f,
    "final_risk": risk_630v18f.shape == expected_shape_630v18f,
}

if not all(shape_checks_630v18f.values()):
    raise RuntimeError("Resident bias retained-array shape changed.")

finite_checks_630v18f = {
    "prediction": bool(np.isfinite(pred_630v18f).all()),
    "mc_std": bool(np.isfinite(std_630v18f).all()),
    "final_risk": bool(np.isfinite(risk_630v18f).all()),
    "mc_std_nonnegative": bool(np.all(std_630v18f >= 0.0)),
    "risk_in_unit_interval": bool(
        np.all((risk_630v18f >= 0.0) & (risk_630v18f <= 1.0))
    ),
}

if not all(finite_checks_630v18f.values()):
    raise RuntimeError("Resident bias retained-array validity check failed.")


# =============================================================================
# 2) RESIDENT BLOCK HASH RECOMPUTATION
# =============================================================================

resident_pred_sha_630v18f = str(
    sha256_array_630v2(pred_630v18f)
)

resident_std_sha_630v18f = str(
    sha256_array_630v2(std_630v18f)
)

resident_risk_sha_630v18f = str(
    sha256_array_630v2(risk_630v18f)
)

resident_hash_checks_630v18f = {
    "prediction":
        resident_pred_sha_630v18f
        == str(FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E),

    "mc_std":
        resident_std_sha_630v18f
        == str(FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E),

    "final_risk":
        resident_risk_sha_630v18f
        == str(FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E),
}

if not all(resident_hash_checks_630v18f.values()):
    raise RuntimeError("Resident bias block hash mismatch.")


# =============================================================================
# 3) SCENARIO MANIFEST / EVIDENCE ORDER
# =============================================================================

bias_manifest_630v18f = (
    FD004_BIAS_SCENARIO_MANIFEST_630V18E
    .copy()
    .reset_index(drop=True)
)

frozen_bias_manifest_630v18f = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U.loc[
        FD004_CORRUPTION_EXECUTION_MANIFEST_630U[
            "family"
        ].astype(str)
        == "bias"
    ]
    .copy()
    .sort_values("scenario_number", kind="mergesort")
    .reset_index(drop=True)
)

if len(frozen_bias_manifest_630v18f) != 102:
    raise RuntimeError(
        "Frozen executable bias manifest no longer has 102 rows."
    )

manifest_id_exact_630v18f = bool(
    np.array_equal(
        bias_manifest_630v18f[
            "scenario_id"
        ].astype(str).to_numpy(),
        frozen_bias_manifest_630v18f[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
)

manifest_number_exact_630v18f = bool(
    np.array_equal(
        bias_manifest_630v18f[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
        frozen_bias_manifest_630v18f[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
    )
)

manifest_family_exact_630v18f = bool(
    np.all(
        bias_manifest_630v18f[
            "family"
        ].astype(str).to_numpy()
        == "bias"
    )
)

if not (
    manifest_id_exact_630v18f
    and manifest_number_exact_630v18f
    and manifest_family_exact_630v18f
):
    raise RuntimeError(
        "Resident bias manifest order/content changed."
    )

scenario_evidence_630v18f = tuple(
    FD004_BIAS_SCENARIO_EVIDENCE_630V18E
)

if len(scenario_evidence_630v18f) != 102:
    raise RuntimeError("Bias scenario evidence count changed.")

evidence_ids_630v18f = np.asarray(
    [
        str(row["scenario_id"])
        for row in scenario_evidence_630v18f
    ],
    dtype=object,
)

evidence_numbers_630v18f = np.asarray(
    [
        int(row["scenario_number"])
        for row in scenario_evidence_630v18f
    ],
    dtype=np.int64,
)

evidence_order_exact_630v18f = bool(
    np.array_equal(
        evidence_ids_630v18f,
        bias_manifest_630v18f[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
    and
    np.array_equal(
        evidence_numbers_630v18f,
        bias_manifest_630v18f[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
    )
)

if not evidence_order_exact_630v18f:
    raise RuntimeError(
        "Bias scenario-evidence ordering changed."
    )

mc_hashes_630v18f = tuple(
    str(x)
    for x in FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E
)

mc_hash_count_exact_630v18f = (
    len(mc_hashes_630v18f) == 102
)

mc_hash_format_exact_630v18f = bool(
    all(
        len(value) == 64
        and all(
            char in "0123456789abcdef"
            for char in value.lower()
        )
        for value in mc_hashes_630v18f
    )
)

if not (
    mc_hash_count_exact_630v18f
    and mc_hash_format_exact_630v18f
):
    raise RuntimeError(
        "Bias full-MC hash evidence changed."
    )


# =============================================================================
# 4) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18f = dict(
    FD004_BIAS_BLOCK_CONTRACT_630V18E
)

recomputed_contract_sha_630v18f = hashlib.sha256(
    json.dumps(
        contract_630v18f,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

contract_sha_exact_630v18f = bool(
    recomputed_contract_sha_630v18f
    ==
    str(FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E)
)

contract_content_checks_630v18f = {
    "family":
        str(contract_630v18f.get("family"))
        == "bias",

    "scenario_count":
        int(contract_630v18f.get("scenario_count"))
        == 102,

    "prediction_hash":
        str(
            contract_630v18f.get(
                "prediction_block_sha256"
            )
        )
        == resident_pred_sha_630v18f,

    "mc_std_hash":
        str(
            contract_630v18f.get(
                "mc_std_block_sha256"
            )
        )
        == resident_std_sha_630v18f,

    "risk_hash":
        str(
            contract_630v18f.get(
                "final_risk_block_sha256"
            )
        )
        == resident_risk_sha_630v18f,

    "dev_outcomes_unused":
        bool(
            contract_630v18f.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18f.get(
                "threshold_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18f.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18f.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not (
    contract_sha_exact_630v18f
    and all(contract_content_checks_630v18f.values())
):
    raise RuntimeError(
        "Bias block contract self-consistency check failed."
    )


# =============================================================================
# 5) CHECKPOINT FILE BYTE HASHES
# =============================================================================

npz_path_630v18f = Path(
    FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E
)

json_path_630v18f = Path(
    FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E
)

if not npz_path_630v18f.is_file():
    raise RuntimeError(
        "Bias NPZ checkpoint file is missing."
    )

if not json_path_630v18f.is_file():
    raise RuntimeError(
        "Bias JSON checkpoint file is missing."
    )


def _file_sha256_630v18f(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


npz_file_sha_630v18f = _file_sha256_630v18f(
    npz_path_630v18f
)

json_file_sha_630v18f = _file_sha256_630v18f(
    json_path_630v18f
)

checkpoint_file_hash_checks_630v18f = {
    "npz":
        npz_file_sha_630v18f
        ==
        str(FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E),

    "json":
        json_file_sha_630v18f
        ==
        str(FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E),
}

if not all(
    checkpoint_file_hash_checks_630v18f.values()
):
    raise RuntimeError(
        "Bias checkpoint file byte hash mismatch."
    )


# =============================================================================
# 6) NPZ ROUND-TRIP EXACTNESS
# =============================================================================

with np.load(
    npz_path_630v18f,
    allow_pickle=False,
) as checkpoint_npz_630v18f:

    checkpoint_keys_630v18f = tuple(
        sorted(checkpoint_npz_630v18f.files)
    )

    expected_checkpoint_keys_630v18f = tuple(
        sorted(
            (
                "scenario_number",
                "scenario_id",
                "deterministic_prediction",
                "mc_std",
                "final_selective_risk",
                "full_mc_matrix_sha256",
            )
        )
    )

    if (
        checkpoint_keys_630v18f
        != expected_checkpoint_keys_630v18f
    ):
        raise RuntimeError(
            "Bias NPZ checkpoint key set changed."
        )

    checkpoint_numbers_630v18f = np.asarray(
        checkpoint_npz_630v18f[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    checkpoint_ids_630v18f = np.asarray(
        checkpoint_npz_630v18f[
            "scenario_id"
        ]
    ).astype(str).reshape(-1)

    checkpoint_pred_630v18f = np.asarray(
        checkpoint_npz_630v18f[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    checkpoint_std_630v18f = np.asarray(
        checkpoint_npz_630v18f[
            "mc_std"
        ],
        dtype=np.float64,
    )

    checkpoint_risk_630v18f = np.asarray(
        checkpoint_npz_630v18f[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    checkpoint_mc_hashes_630v18f = tuple(
        str(x)
        for x in np.asarray(
            checkpoint_npz_630v18f[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


npz_roundtrip_checks_630v18f = {
    "scenario_number":
        bool(
            np.array_equal(
                checkpoint_numbers_630v18f,
                bias_manifest_630v18f[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                checkpoint_ids_630v18f,
                bias_manifest_630v18f[
                    "scenario_id"
                ].astype(str).to_numpy(),
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                checkpoint_pred_630v18f,
                pred_630v18f,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                checkpoint_std_630v18f,
                std_630v18f,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                checkpoint_risk_630v18f,
                risk_630v18f,
            )
        ),

    "full_mc_hashes":
        checkpoint_mc_hashes_630v18f
        == mc_hashes_630v18f,
}

if not all(npz_roundtrip_checks_630v18f.values()):
    raise RuntimeError(
        "Bias NPZ checkpoint round-trip mismatch."
    )


# =============================================================================
# 7) JSON CHECKPOINT ROUND-TRIP
# =============================================================================

with open(
    json_path_630v18f,
    "r",
    encoding="utf-8",
) as file:
    checkpoint_json_630v18f = json.load(file)

json_checks_630v18f = {
    "dataset":
        str(checkpoint_json_630v18f.get("dataset"))
        == "FD004",

    "stage":
        str(checkpoint_json_630v18f.get("stage"))
        == "6.30V-18E",

    "family":
        str(checkpoint_json_630v18f.get("family"))
        == "bias",

    "scenario_count":
        int(
            checkpoint_json_630v18f.get(
                "scenario_count"
            )
        )
        == 102,

    "block_contract_sha":
        str(
            checkpoint_json_630v18f.get(
                "block_contract_sha256"
            )
        )
        ==
        str(FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E),

    "prediction_sha":
        str(
            checkpoint_json_630v18f.get(
                "prediction_block_sha256"
            )
        )
        == resident_pred_sha_630v18f,

    "mc_std_sha":
        str(
            checkpoint_json_630v18f.get(
                "mc_std_block_sha256"
            )
        )
        == resident_std_sha_630v18f,

    "risk_sha":
        str(
            checkpoint_json_630v18f.get(
                "final_risk_block_sha256"
            )
        )
        == resident_risk_sha_630v18f,

    "scenario_evidence_count":
        len(
            checkpoint_json_630v18f.get(
                "scenario_evidence",
                [],
            )
        )
        == 102,

    "dev_outcomes_unused":
        bool(
            checkpoint_json_630v18f.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            checkpoint_json_630v18f.get(
                "threshold_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            checkpoint_json_630v18f.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            checkpoint_json_630v18f.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(json_checks_630v18f.values()):
    raise RuntimeError(
        "Bias JSON checkpoint round-trip mismatch."
    )


# =============================================================================
# 8) FREEZE AUDIT
# =============================================================================

FD004_BIAS_BLOCK_EVIDENCE_AUDIT_630V18F = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18F",

    "family":
        "bias",

    "status":
        "BIAS_BLOCK_EVIDENCE_AND_CHECKPOINT_VERIFIED",

    "scenario_count":
        102,

    "retained_shape":
        expected_shape_630v18f,

    "resident_hash_checks":
        resident_hash_checks_630v18f,

    "manifest_id_exact":
        manifest_id_exact_630v18f,

    "manifest_number_exact":
        manifest_number_exact_630v18f,

    "scenario_evidence_order_exact":
        evidence_order_exact_630v18f,

    "full_mc_hash_count_exact":
        mc_hash_count_exact_630v18f,

    "full_mc_hash_format_exact":
        mc_hash_format_exact_630v18f,

    "block_contract_sha_exact":
        contract_sha_exact_630v18f,

    "checkpoint_file_hash_checks":
        checkpoint_file_hash_checks_630v18f,

    "npz_roundtrip_checks":
        npz_roundtrip_checks_630v18f,

    "json_roundtrip_checks":
        json_checks_630v18f,

    "scientific_execution_performed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_BIAS_BLOCK_EVIDENCE_AUDIT_SHA256_630V18F = (
    hashlib.sha256(
        json.dumps(
            FD004_BIAS_BLOCK_EVIDENCE_AUDIT_630V18F,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_BIAS_BLOCK_EVIDENCE_VERIFIED_630V18F = True
FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18F = False
FD004_DEV_TARGETS_ERRORS_USED_630V18F = False
FD004_THRESHOLD_SELECTED_630V18F = False
FD004_POLICY_SELECTED_630V18F = False
FD004_TEST_CONTENTS_READ_630V18F = False
FD004_RUL_CONTENTS_READ_630V18F = False

all_checks_passed_630v18f = True


# =============================================================================
# 9) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18F — FD004 BIAS BLOCK "
    "EVIDENCE + CHECKPOINT ROUND-TRIP AUDIT"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- RESIDENT BLOCK ---")

print(
    "Retained array shape              :",
    expected_shape_630v18f,
)

print(
    "Prediction SHA exact              :",
    resident_hash_checks_630v18f[
        "prediction"
    ],
)

print(
    "MC STD SHA exact                  :",
    resident_hash_checks_630v18f[
        "mc_std"
    ],
)

print(
    "Final risk SHA exact              :",
    resident_hash_checks_630v18f[
        "final_risk"
    ],
)

print(
    "Manifest scenario IDs exact       :",
    manifest_id_exact_630v18f,
)

print(
    "Manifest scenario numbers exact   :",
    manifest_number_exact_630v18f,
)

print(
    "Scenario evidence order exact     :",
    evidence_order_exact_630v18f,
)

print(
    "Full-MC hash count / format exact :",
    (
        mc_hash_count_exact_630v18f,
        mc_hash_format_exact_630v18f,
    ),
)

print(
    "Block contract SHA exact          :",
    contract_sha_exact_630v18f,
)


print("\n--- CHECKPOINT FILE INTEGRITY ---")

print(
    "NPZ byte SHA exact                :",
    checkpoint_file_hash_checks_630v18f[
        "npz"
    ],
)

print(
    "JSON byte SHA exact               :",
    checkpoint_file_hash_checks_630v18f[
        "json"
    ],
)


print("\n--- NPZ ROUND-TRIP ---")

for key, value in npz_roundtrip_checks_630v18f.items():
    print(
        f"{key:32s}:",
        value,
    )


print("\n--- JSON ROUND-TRIP ---")

for key, value in json_checks_630v18f.items():
    print(
        f"{key:32s}:",
        value,
    )


print("\n--- AUDIT LOCK ---")

print(
    "Bias evidence verified            :",
    FD004_BIAS_BLOCK_EVIDENCE_VERIFIED_630V18F,
)

print(
    "Bias evidence audit SHA-256       :",
    FD004_BIAS_BLOCK_EVIDENCE_AUDIT_SHA256_630V18F,
)

print(
    "\nPASS — the completed FD004 bias block is resident-hash exact "
    "and its NPZ/JSON checkpoint round-trips exactly."
)

print(
    "Next: execute only the frozen 51-scenario noise block using "
    "the same verified label-free execution path."
)

In [ ]:
# STEP 6.30V-18F-R1 — FD004 BIAS CHECKPOINT SAFE-SERIALIZATION REPAIR
#
# PERSISTENCE REPAIR ONLY.
#
# NO corruption.
# NO normalization/windows.
# NO deterministic model inference.
# NO MC.
# NO reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
#
# IMPORTANT:
# - Original v1 checkpoint is NOT overwritten.
# - v1 is first authenticated by its frozen SHA-256.
# - Only after authentication is allow_pickle=True used to read
#   the known self-created object-string array.
# - A v2 checkpoint is then written with fixed-width Unicode strings.
# - v2 MUST load successfully with allow_pickle=False.

import numpy as np
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18fr1 = (
    "all_checks_passed_630v18e",
    "FD004_BIAS_BLOCK_COMPLETE_630V18E",
    "FD004_BIAS_SCENARIOS_EXECUTED_630V18E",
    "FD004_BIAS_SCENARIO_MANIFEST_630V18E",
    "FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E",
    "FD004_BIAS_MC_STD_630V18E",
    "FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E",
    "FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E",
    "FD004_BIAS_SCENARIO_EVIDENCE_630V18E",
    "FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E",
    "FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E",
    "FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E",
    "FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E",
    "FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E",
    "FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E",
    "FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E",
    "FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E",
    "sha256_array_630v2",
)

missing_630v18fr1 = [
    name
    for name in required_630v18fr1
    if name not in globals()
]

if missing_630v18fr1:
    raise RuntimeError(
        "Missing STEP 6.30V-18F-R1 authority object(s): "
        + ", ".join(missing_630v18fr1)
    )

if not bool(all_checks_passed_630v18e):
    raise RuntimeError(
        "STEP 6.30V-18E must remain passed."
    )

if not bool(FD004_BIAS_BLOCK_COMPLETE_630V18E):
    raise RuntimeError(
        "Bias block is not marked complete."
    )

if int(FD004_BIAS_SCENARIOS_EXECUTED_630V18E) != 102:
    raise RuntimeError(
        "Bias scenario count changed."
    )


# =============================================================================
# 1) RESIDENT SCIENTIFIC EVIDENCE
# =============================================================================

manifest_630v18fr1 = (
    FD004_BIAS_SCENARIO_MANIFEST_630V18E
    .copy()
    .reset_index(drop=True)
)

pred_630v18fr1 = np.asarray(
    FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E,
    dtype=np.float32,
)

std_630v18fr1 = np.asarray(
    FD004_BIAS_MC_STD_630V18E,
    dtype=np.float64,
)

risk_630v18fr1 = np.asarray(
    FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E,
    dtype=np.float64,
)

mc_hashes_630v18fr1 = tuple(
    str(x)
    for x in FD004_BIAS_FULL_MC_MATRIX_SHA256_630V18E
)

scenario_numbers_630v18fr1 = (
    manifest_630v18fr1[
        "scenario_number"
    ]
    .to_numpy(dtype=np.int64)
)

scenario_ids_list_630v18fr1 = (
    manifest_630v18fr1[
        "scenario_id"
    ]
    .astype(str)
    .tolist()
)

max_id_length_630v18fr1 = max(
    len(value)
    for value in scenario_ids_list_630v18fr1
)

# CRITICAL REPAIR:
# fixed-width Unicode, never dtype=object.
scenario_ids_unicode_630v18fr1 = np.asarray(
    scenario_ids_list_630v18fr1,
    dtype=f"<U{max_id_length_630v18fr1}",
)

mc_hashes_unicode_630v18fr1 = np.asarray(
    mc_hashes_630v18fr1,
    dtype="<U64",
)

if pred_630v18fr1.shape != (102, 10824):
    raise RuntimeError(
        "Resident bias prediction shape changed."
    )

if std_630v18fr1.shape != (102, 10824):
    raise RuntimeError(
        "Resident bias MC-STD shape changed."
    )

if risk_630v18fr1.shape != (102, 10824):
    raise RuntimeError(
        "Resident bias risk shape changed."
    )

if len(mc_hashes_630v18fr1) != 102:
    raise RuntimeError(
        "Resident full-MC hash count changed."
    )


# =============================================================================
# 2) RESIDENT HASHES MUST STILL MATCH FROZEN 18E EVIDENCE
# =============================================================================

resident_pred_sha_630v18fr1 = str(
    sha256_array_630v2(
        pred_630v18fr1
    )
)

resident_std_sha_630v18fr1 = str(
    sha256_array_630v2(
        std_630v18fr1
    )
)

resident_risk_sha_630v18fr1 = str(
    sha256_array_630v2(
        risk_630v18fr1
    )
)

resident_hash_checks_630v18fr1 = {
    "prediction":
        resident_pred_sha_630v18fr1
        ==
        str(
            FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E
        ),

    "mc_std":
        resident_std_sha_630v18fr1
        ==
        str(
            FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E
        ),

    "final_risk":
        resident_risk_sha_630v18fr1
        ==
        str(
            FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E
        ),
}

if not all(
    resident_hash_checks_630v18fr1.values()
):
    raise RuntimeError(
        "Resident scientific bias evidence no longer matches "
        "the frozen 18E hashes."
    )


# =============================================================================
# 3) FILE-HASH HELPER
# =============================================================================

def _file_sha256_630v18fr1(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 4) AUTHENTICATE ORIGINAL v1 CHECKPOINT BEFORE allow_pickle=True
# =============================================================================

v1_npz_path_630v18fr1 = Path(
    FD004_BIAS_CHECKPOINT_NPZ_PATH_630V18E
)

v1_json_path_630v18fr1 = Path(
    FD004_BIAS_CHECKPOINT_JSON_PATH_630V18E
)

if not v1_npz_path_630v18fr1.is_file():
    raise RuntimeError(
        "Original v1 bias NPZ checkpoint is missing."
    )

if not v1_json_path_630v18fr1.is_file():
    raise RuntimeError(
        "Original v1 bias JSON checkpoint is missing."
    )

v1_npz_sha_630v18fr1 = (
    _file_sha256_630v18fr1(
        v1_npz_path_630v18fr1
    )
)

v1_json_sha_630v18fr1 = (
    _file_sha256_630v18fr1(
        v1_json_path_630v18fr1
    )
)

v1_npz_authenticated_630v18fr1 = bool(
    v1_npz_sha_630v18fr1
    ==
    str(
        FD004_BIAS_CHECKPOINT_NPZ_SHA256_630V18E
    )
)

v1_json_authenticated_630v18fr1 = bool(
    v1_json_sha_630v18fr1
    ==
    str(
        FD004_BIAS_CHECKPOINT_JSON_SHA256_630V18E
    )
)

if not (
    v1_npz_authenticated_630v18fr1
    and
    v1_json_authenticated_630v18fr1
):
    raise RuntimeError(
        "Original v1 checkpoint failed frozen byte-hash authentication. "
        "It will not be loaded with allow_pickle=True."
    )


# =============================================================================
# 5) READ AUTHENTICATED v1 ONCE
#
# allow_pickle=True is permitted ONLY here because:
# 1. this file was created by STEP 18E itself, and
# 2. its entire-file SHA-256 has just matched the frozen recorded SHA.
# =============================================================================

expected_keys_630v18fr1 = tuple(
    sorted(
        (
            "scenario_number",
            "scenario_id",
            "deterministic_prediction",
            "mc_std",
            "final_selective_risk",
            "full_mc_matrix_sha256",
        )
    )
)

with np.load(
    v1_npz_path_630v18fr1,
    allow_pickle=True,
) as v1_npz_630v18fr1:

    v1_keys_630v18fr1 = tuple(
        sorted(
            v1_npz_630v18fr1.files
        )
    )

    if (
        v1_keys_630v18fr1
        !=
        expected_keys_630v18fr1
    ):
        raise RuntimeError(
            "Authenticated v1 checkpoint key set changed."
        )

    v1_numbers_630v18fr1 = np.asarray(
        v1_npz_630v18fr1[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    v1_scenario_raw_630v18fr1 = np.asarray(
        v1_npz_630v18fr1[
            "scenario_id"
        ]
    )

    v1_scenario_dtype_630v18fr1 = str(
        v1_scenario_raw_630v18fr1.dtype
    )

    v1_scenario_has_object_dtype_630v18fr1 = bool(
        v1_scenario_raw_630v18fr1.dtype.hasobject
    )

    v1_ids_630v18fr1 = (
        v1_scenario_raw_630v18fr1
        .astype(str)
        .reshape(-1)
    )

    v1_pred_630v18fr1 = np.asarray(
        v1_npz_630v18fr1[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    v1_std_630v18fr1 = np.asarray(
        v1_npz_630v18fr1[
            "mc_std"
        ],
        dtype=np.float64,
    )

    v1_risk_630v18fr1 = np.asarray(
        v1_npz_630v18fr1[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    v1_mc_hashes_630v18fr1 = tuple(
        str(x)
        for x in np.asarray(
            v1_npz_630v18fr1[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


# =============================================================================
# 6) v1 MUST MATCH RESIDENT SCIENTIFIC EVIDENCE EXACTLY
# =============================================================================

v1_roundtrip_checks_630v18fr1 = {
    "scenario_number":
        bool(
            np.array_equal(
                v1_numbers_630v18fr1,
                scenario_numbers_630v18fr1,
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                v1_ids_630v18fr1,
                scenario_ids_unicode_630v18fr1,
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                v1_pred_630v18fr1,
                pred_630v18fr1,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                v1_std_630v18fr1,
                std_630v18fr1,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                v1_risk_630v18fr1,
                risk_630v18fr1,
            )
        ),

    "full_mc_hashes":
        v1_mc_hashes_630v18fr1
        ==
        mc_hashes_630v18fr1,
}

if not all(
    v1_roundtrip_checks_630v18fr1.values()
):
    raise RuntimeError(
        "Authenticated v1 checkpoint does not exactly match "
        "resident scientific bias evidence."
    )


# =============================================================================
# 7) ORIGINAL v1 JSON MUST ALSO MATCH FROZEN EVIDENCE
# =============================================================================

with open(
    v1_json_path_630v18fr1,
    "r",
    encoding="utf-8",
) as file:
    v1_json_630v18fr1 = json.load(
        file
    )

v1_json_checks_630v18fr1 = {
    "dataset":
        str(
            v1_json_630v18fr1.get(
                "dataset"
            )
        )
        ==
        "FD004",

    "stage":
        str(
            v1_json_630v18fr1.get(
                "stage"
            )
        )
        ==
        "6.30V-18E",

    "family":
        str(
            v1_json_630v18fr1.get(
                "family"
            )
        )
        ==
        "bias",

    "scenario_count":
        int(
            v1_json_630v18fr1.get(
                "scenario_count"
            )
        )
        ==
        102,

    "block_contract_sha":
        str(
            v1_json_630v18fr1.get(
                "block_contract_sha256"
            )
        )
        ==
        str(
            FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E
        ),

    "prediction_sha":
        str(
            v1_json_630v18fr1.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_pred_sha_630v18fr1,

    "mc_std_sha":
        str(
            v1_json_630v18fr1.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_std_sha_630v18fr1,

    "risk_sha":
        str(
            v1_json_630v18fr1.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_risk_sha_630v18fr1,

    "scenario_evidence_count":
        len(
            v1_json_630v18fr1.get(
                "scenario_evidence",
                [],
            )
        )
        ==
        102,

    "dev_outcomes_unused":
        bool(
            v1_json_630v18fr1.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            v1_json_630v18fr1.get(
                "threshold_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            v1_json_630v18fr1.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            v1_json_630v18fr1.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(
    v1_json_checks_630v18fr1.values()
):
    raise RuntimeError(
        "Original v1 JSON checkpoint does not match frozen evidence."
    )


# =============================================================================
# 8) WRITE SAFE v2 CHECKPOINT
#
# Original v1 is deliberately preserved.
# =============================================================================

checkpoint_dir_630v18fr1 = (
    v1_npz_path_630v18fr1.parent
)

v2_npz_path_630v18fr1 = (
    checkpoint_dir_630v18fr1
    /
    "fd004_bias_block_630v18e_v2_safe.npz"
)

v2_json_path_630v18fr1 = (
    checkpoint_dir_630v18fr1
    /
    "fd004_bias_block_630v18e_v2_safe.json"
)

v2_npz_tmp_630v18fr1 = (
    checkpoint_dir_630v18fr1
    /
    "fd004_bias_block_630v18e_v2_safe.npz.tmp"
)

v2_json_tmp_630v18fr1 = (
    checkpoint_dir_630v18fr1
    /
    "fd004_bias_block_630v18e_v2_safe.json.tmp"
)


# Write NPZ through an open binary file so NumPy does not append ".npz".
with open(
    v2_npz_tmp_630v18fr1,
    "wb",
) as file:

    np.savez_compressed(
        file,
        scenario_number=
            scenario_numbers_630v18fr1,

        scenario_id=
            scenario_ids_unicode_630v18fr1,

        deterministic_prediction=
            pred_630v18fr1,

        mc_std=
            std_630v18fr1,

        final_selective_risk=
            risk_630v18fr1,

        full_mc_matrix_sha256=
            mc_hashes_unicode_630v18fr1,
    )

os.replace(
    v2_npz_tmp_630v18fr1,
    v2_npz_path_630v18fr1,
)

v2_npz_sha_630v18fr1 = (
    _file_sha256_630v18fr1(
        v2_npz_path_630v18fr1
    )
)


v2_metadata_630v18fr1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18F-R1",

    "family":
        "bias",

    "status":
        "SAFE_SERIALIZATION_CHECKPOINT",

    "scenario_count":
        102,

    "repair_scope":
        "persistence_only",

    "scientific_results_changed":
        False,

    "repair_reason":
        (
            "v1 scenario_id was serialized as a NumPy object array; "
            "v2 stores scenario_id as fixed-width Unicode so the NPZ "
            "can be loaded with allow_pickle=False"
        ),

    "source_v1_npz_path":
        str(
            v1_npz_path_630v18fr1
        ),

    "source_v1_npz_sha256":
        v1_npz_sha_630v18fr1,

    "source_v1_json_path":
        str(
            v1_json_path_630v18fr1
        ),

    "source_v1_json_sha256":
        v1_json_sha_630v18fr1,

    "source_v1_scenario_id_dtype":
        v1_scenario_dtype_630v18fr1,

    "source_v1_scenario_id_has_object_dtype":
        v1_scenario_has_object_dtype_630v18fr1,

    "v2_scenario_id_dtype":
        str(
            scenario_ids_unicode_630v18fr1.dtype
        ),

    "v2_npz_sha256":
        v2_npz_sha_630v18fr1,

    "block_contract_sha256":
        str(
            FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E
        ),

    "prediction_block_sha256":
        resident_pred_sha_630v18fr1,

    "mc_std_block_sha256":
        resident_std_sha_630v18fr1,

    "final_risk_block_sha256":
        resident_risk_sha_630v18fr1,

    "scenario_evidence":
        list(
            FD004_BIAS_SCENARIO_EVIDENCE_630V18E
        ),

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


with open(
    v2_json_tmp_630v18fr1,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        v2_metadata_630v18fr1,
        file,
        sort_keys=True,
        indent=2,
        default=str,
    )

os.replace(
    v2_json_tmp_630v18fr1,
    v2_json_path_630v18fr1,
)

v2_json_sha_630v18fr1 = (
    _file_sha256_630v18fr1(
        v2_json_path_630v18fr1
    )
)


# =============================================================================
# 9) VERIFY v2 WITH allow_pickle=False — THIS IS THE REQUIRED SAFE PATH
# =============================================================================

with np.load(
    v2_npz_path_630v18fr1,
    allow_pickle=False,
) as v2_npz_630v18fr1:

    v2_keys_630v18fr1 = tuple(
        sorted(
            v2_npz_630v18fr1.files
        )
    )

    if (
        v2_keys_630v18fr1
        !=
        expected_keys_630v18fr1
    ):
        raise RuntimeError(
            "v2 checkpoint key set changed."
        )

    v2_numbers_630v18fr1 = np.asarray(
        v2_npz_630v18fr1[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    v2_scenario_raw_630v18fr1 = np.asarray(
        v2_npz_630v18fr1[
            "scenario_id"
        ]
    )

    v2_scenario_dtype_630v18fr1 = str(
        v2_scenario_raw_630v18fr1.dtype
    )

    v2_scenario_has_object_dtype_630v18fr1 = bool(
        v2_scenario_raw_630v18fr1.dtype.hasobject
    )

    v2_ids_630v18fr1 = (
        v2_scenario_raw_630v18fr1
        .astype(str)
        .reshape(-1)
    )

    v2_pred_630v18fr1 = np.asarray(
        v2_npz_630v18fr1[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    v2_std_630v18fr1 = np.asarray(
        v2_npz_630v18fr1[
            "mc_std"
        ],
        dtype=np.float64,
    )

    v2_risk_630v18fr1 = np.asarray(
        v2_npz_630v18fr1[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    v2_mc_hashes_630v18fr1 = tuple(
        str(x)
        for x in np.asarray(
            v2_npz_630v18fr1[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


v2_safe_dtype_check_630v18fr1 = bool(
    not v2_scenario_has_object_dtype_630v18fr1
    and
    v2_scenario_raw_630v18fr1.dtype.kind
    in (
        "U",
        "S",
    )
)

v2_roundtrip_checks_630v18fr1 = {
    "safe_non_object_scenario_id_dtype":
        v2_safe_dtype_check_630v18fr1,

    "scenario_number":
        bool(
            np.array_equal(
                v2_numbers_630v18fr1,
                scenario_numbers_630v18fr1,
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                v2_ids_630v18fr1,
                scenario_ids_unicode_630v18fr1,
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                v2_pred_630v18fr1,
                pred_630v18fr1,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                v2_std_630v18fr1,
                std_630v18fr1,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                v2_risk_630v18fr1,
                risk_630v18fr1,
            )
        ),

    "full_mc_hashes":
        v2_mc_hashes_630v18fr1
        ==
        mc_hashes_630v18fr1,
}

if not all(
    v2_roundtrip_checks_630v18fr1.values()
):
    failed_630v18fr1 = [
        key
        for key, passed
        in v2_roundtrip_checks_630v18fr1.items()
        if not passed
    ]

    raise RuntimeError(
        "Safe v2 checkpoint round-trip failed: "
        + ", ".join(
            failed_630v18fr1
        )
    )


# =============================================================================
# 10) VERIFY v2 JSON
# =============================================================================

with open(
    v2_json_path_630v18fr1,
    "r",
    encoding="utf-8",
) as file:
    v2_json_loaded_630v18fr1 = json.load(
        file
    )

v2_json_checks_630v18fr1 = {
    "dataset":
        v2_json_loaded_630v18fr1.get(
            "dataset"
        )
        ==
        "FD004",

    "stage":
        v2_json_loaded_630v18fr1.get(
            "stage"
        )
        ==
        "6.30V-18F-R1",

    "family":
        v2_json_loaded_630v18fr1.get(
            "family"
        )
        ==
        "bias",

    "scenario_count":
        int(
            v2_json_loaded_630v18fr1.get(
                "scenario_count"
            )
        )
        ==
        102,

    "scientific_results_unchanged":
        bool(
            v2_json_loaded_630v18fr1.get(
                "scientific_results_changed"
            )
        )
        is False,

    "v1_npz_sha":
        str(
            v2_json_loaded_630v18fr1.get(
                "source_v1_npz_sha256"
            )
        )
        ==
        v1_npz_sha_630v18fr1,

    "v2_npz_sha":
        str(
            v2_json_loaded_630v18fr1.get(
                "v2_npz_sha256"
            )
        )
        ==
        v2_npz_sha_630v18fr1,

    "prediction_sha":
        str(
            v2_json_loaded_630v18fr1.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_pred_sha_630v18fr1,

    "mc_std_sha":
        str(
            v2_json_loaded_630v18fr1.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_std_sha_630v18fr1,

    "risk_sha":
        str(
            v2_json_loaded_630v18fr1.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_risk_sha_630v18fr1,

    "dev_outcomes_unused":
        bool(
            v2_json_loaded_630v18fr1.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            v2_json_loaded_630v18fr1.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            v2_json_loaded_630v18fr1.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            v2_json_loaded_630v18fr1.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            v2_json_loaded_630v18fr1.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(
    v2_json_checks_630v18fr1.values()
):
    raise RuntimeError(
        "Safe v2 JSON verification failed."
    )


# =============================================================================
# 11) VERIFY ORIGINAL v1 WAS NOT MODIFIED
# =============================================================================

v1_npz_sha_after_630v18fr1 = (
    _file_sha256_630v18fr1(
        v1_npz_path_630v18fr1
    )
)

v1_json_sha_after_630v18fr1 = (
    _file_sha256_630v18fr1(
        v1_json_path_630v18fr1
    )
)

v1_preserved_exact_630v18fr1 = bool(
    v1_npz_sha_after_630v18fr1
    ==
    v1_npz_sha_630v18fr1
    and
    v1_json_sha_after_630v18fr1
    ==
    v1_json_sha_630v18fr1
)

if not v1_preserved_exact_630v18fr1:
    raise RuntimeError(
        "Original v1 checkpoint changed during repair."
    )


# =============================================================================
# 12) FREEZE REPAIR EVIDENCE
# =============================================================================

FD004_BIAS_SAFE_CHECKPOINT_NPZ_PATH_630V18FR1 = str(
    v2_npz_path_630v18fr1
)

FD004_BIAS_SAFE_CHECKPOINT_JSON_PATH_630V18FR1 = str(
    v2_json_path_630v18fr1
)

FD004_BIAS_SAFE_CHECKPOINT_NPZ_SHA256_630V18FR1 = (
    v2_npz_sha_630v18fr1
)

FD004_BIAS_SAFE_CHECKPOINT_JSON_SHA256_630V18FR1 = (
    v2_json_sha_630v18fr1
)

FD004_BIAS_SAFE_CHECKPOINT_REPAIR_CONTRACT_630V18FR1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18F-R1",

    "status":
        "BIAS_CHECKPOINT_SAFE_SERIALIZATION_REPAIRED_AND_VERIFIED",

    "scientific_execution_performed":
        False,

    "scientific_results_changed":
        False,

    "original_v1_preserved_exact":
        v1_preserved_exact_630v18fr1,

    "v1_npz_authenticated":
        v1_npz_authenticated_630v18fr1,

    "v1_json_authenticated":
        v1_json_authenticated_630v18fr1,

    "v1_scenario_id_dtype":
        v1_scenario_dtype_630v18fr1,

    "v1_scenario_id_has_object_dtype":
        v1_scenario_has_object_dtype_630v18fr1,

    "v1_roundtrip_checks":
        v1_roundtrip_checks_630v18fr1,

    "v1_json_checks":
        v1_json_checks_630v18fr1,

    "v2_scenario_id_dtype":
        v2_scenario_dtype_630v18fr1,

    "v2_scenario_id_has_object_dtype":
        v2_scenario_has_object_dtype_630v18fr1,

    "v2_allow_pickle_false_roundtrip":
        v2_roundtrip_checks_630v18fr1,

    "v2_json_checks":
        v2_json_checks_630v18fr1,

    "v2_npz_sha256":
        v2_npz_sha_630v18fr1,

    "v2_json_sha256":
        v2_json_sha_630v18fr1,

    "prediction_block_sha256":
        resident_pred_sha_630v18fr1,

    "mc_std_block_sha256":
        resident_std_sha_630v18fr1,

    "final_risk_block_sha256":
        resident_risk_sha_630v18fr1,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_BIAS_SAFE_CHECKPOINT_REPAIR_SHA256_630V18FR1 = (
    hashlib.sha256(
        json.dumps(
            FD004_BIAS_SAFE_CHECKPOINT_REPAIR_CONTRACT_630V18FR1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_BIAS_CHECKPOINT_SAFE_630V18FR1 = True
FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18FR1 = False

FD004_DEV_TARGETS_ERRORS_USED_630V18FR1 = False
FD004_THRESHOLD_SELECTED_630V18FR1 = False
FD004_POLICY_SELECTED_630V18FR1 = False
FD004_TEST_CONTENTS_READ_630V18FR1 = False
FD004_RUL_CONTENTS_READ_630V18FR1 = False

all_checks_passed_630v18fr1 = True


# =============================================================================
# 13) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18F-R1 — FD004 BIAS CHECKPOINT "
    "SAFE-SERIALIZATION REPAIR"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Bias scenarios rerun              :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- ORIGINAL v1 AUTHENTICATION ---")

print(
    "v1 NPZ frozen SHA exact           :",
    v1_npz_authenticated_630v18fr1,
)

print(
    "v1 JSON frozen SHA exact          :",
    v1_json_authenticated_630v18fr1,
)

print(
    "v1 scenario_id dtype              :",
    v1_scenario_dtype_630v18fr1,
)

print(
    "v1 scenario_id is object dtype    :",
    v1_scenario_has_object_dtype_630v18fr1,
)

print(
    "Original v1 preserved exactly     :",
    v1_preserved_exact_630v18fr1,
)


print("\n--- AUTHENTICATED v1 ROUND-TRIP ---")

for key, value in (
    v1_roundtrip_checks_630v18fr1.items()
):
    print(
        f"{key:36s}:",
        value,
    )


print("\n--- SAFE v2 SERIALIZATION ---")

print(
    "v2 scenario_id dtype              :",
    v2_scenario_dtype_630v18fr1,
)

print(
    "v2 scenario_id is object dtype    :",
    v2_scenario_has_object_dtype_630v18fr1,
)

print(
    "v2 loaded with allow_pickle=False :",
    True,
)


print("\n--- SAFE v2 ROUND-TRIP ---")

for key, value in (
    v2_roundtrip_checks_630v18fr1.items()
):
    print(
        f"{key:36s}:",
        value,
    )


print("\n--- SCIENTIFIC BLOCK HASHES ---")

print(
    "Prediction block SHA exact        :",
    resident_hash_checks_630v18fr1[
        "prediction"
    ],
)

print(
    "MC STD block SHA exact            :",
    resident_hash_checks_630v18fr1[
        "mc_std"
    ],
)

print(
    "Final risk block SHA exact        :",
    resident_hash_checks_630v18fr1[
        "final_risk"
    ],
)


print("\n--- SAFE CHECKPOINT ---")

print(
    "v2 NPZ path                      :",
    FD004_BIAS_SAFE_CHECKPOINT_NPZ_PATH_630V18FR1,
)

print(
    "v2 JSON path                     :",
    FD004_BIAS_SAFE_CHECKPOINT_JSON_PATH_630V18FR1,
)

print(
    "v2 NPZ SHA-256                   :",
    FD004_BIAS_SAFE_CHECKPOINT_NPZ_SHA256_630V18FR1,
)

print(
    "v2 JSON SHA-256                  :",
    FD004_BIAS_SAFE_CHECKPOINT_JSON_SHA256_630V18FR1,
)

print(
    "Repair contract SHA-256          :",
    FD004_BIAS_SAFE_CHECKPOINT_REPAIR_SHA256_630V18FR1,
)


print(
    "\nPASS — the bias scientific results were NOT rerun or changed. "
    "The original authenticated v1 checkpoint was preserved, and a "
    "pickle-free v2 checkpoint now round-trips exactly."
)

print(
    "Next: perform the final bias evidence closure from the verified "
    "v2 checkpoint, then proceed to the 51-scenario noise block."
)

In [ ]:
# STEP 6.30V-18G — FD004 BIAS FAMILY FINAL EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
# This cell only closes already-verified bias evidence.
#
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.

import hashlib
import json
from pathlib import Path

# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18g = (
    "all_checks_passed_630v18e",
    "all_checks_passed_630v18fr1",
    "FD004_BIAS_BLOCK_COMPLETE_630V18E",
    "FD004_BIAS_SCENARIOS_EXECUTED_630V18E",
    "FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E",
    "FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E",
    "FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E",
    "FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E",
    "FD004_BIAS_SAFE_CHECKPOINT_REPAIR_SHA256_630V18FR1",
    "FD004_BIAS_SAFE_CHECKPOINT_NPZ_PATH_630V18FR1",
    "FD004_BIAS_SAFE_CHECKPOINT_JSON_PATH_630V18FR1",
    "FD004_BIAS_SAFE_CHECKPOINT_NPZ_SHA256_630V18FR1",
    "FD004_BIAS_SAFE_CHECKPOINT_JSON_SHA256_630V18FR1",
    "FD004_BIAS_SAFE_CHECKPOINT_REPAIR_CONTRACT_630V18FR1",
)

missing_630v18g = [
    name_630v18g
    for name_630v18g in required_630v18g
    if name_630v18g not in globals()
]

if missing_630v18g:
    raise RuntimeError(
        "Missing STEP 6.30V-18G authority object(s): "
        + ", ".join(missing_630v18g)
    )

if not (
    bool(all_checks_passed_630v18e)
    and
    bool(all_checks_passed_630v18fr1)
):
    raise RuntimeError(
        "Bias execution and checkpoint-repair stages must remain passed."
    )

# =============================================================================
# 1) EXACT FROZEN BIAS EVIDENCE GUARDS
# =============================================================================

expected_630v18g = {
    "prediction":
        "3811e0761dfd3aeae80cfa07bbddd6ee1687b345de9e84341a3eba8e9f5d709c",

    "mc_std":
        "aae813003499f1182fabea81feb898a709a14bf88c6c65e83cdaf29da4bd7a7e",

    "final_risk":
        "70e60923feee12eb2bd7fd382b4a9cf88785a9ada1ead279cbed21d1d18fd788",

    "bias_block_contract":
        "2605777023cb74b428c051a9ac7294b99f9a4f46c1580679568af84945e9ba5a",

    "safe_checkpoint_repair":
        "de9c83f66dd11229cbcaad85f2e26545b48b4a84711691cd65df3c5ec9c91498",

    "safe_npz":
        "338075544290ccdee1ee8ab0f63dbed7af576ae45e2a2f2fe6a2be71d95525c8",

    "safe_json":
        "038b3b2196af987e6c2024975f6f4a444d9fe8aeef8b20a12da7bedbb9606b25",
}

observed_630v18g = {
    "prediction":
        str(FD004_BIAS_BLOCK_PREDICTIONS_SHA256_630V18E),

    "mc_std":
        str(FD004_BIAS_BLOCK_MC_STD_SHA256_630V18E),

    "final_risk":
        str(FD004_BIAS_BLOCK_FINAL_RISK_SHA256_630V18E),

    "bias_block_contract":
        str(FD004_BIAS_BLOCK_CONTRACT_SHA256_630V18E),

    "safe_checkpoint_repair":
        str(FD004_BIAS_SAFE_CHECKPOINT_REPAIR_SHA256_630V18FR1),

    "safe_npz":
        str(FD004_BIAS_SAFE_CHECKPOINT_NPZ_SHA256_630V18FR1),

    "safe_json":
        str(FD004_BIAS_SAFE_CHECKPOINT_JSON_SHA256_630V18FR1),
}

hash_checks_630v18g = {
    key_630v18g:
        observed_630v18g[key_630v18g]
        ==
        expected_630v18g[key_630v18g]
    for key_630v18g
    in expected_630v18g
}

if not all(hash_checks_630v18g.values()):
    failed_630v18g = [
        key_630v18g
        for key_630v18g, passed_630v18g
        in hash_checks_630v18g.items()
        if not passed_630v18g
    ]

    raise RuntimeError(
        "Frozen bias evidence changed: "
        + ", ".join(failed_630v18g)
    )

# =============================================================================
# 2) VERIFIED SAFE-CHECKPOINT CONTRACT
# =============================================================================

repair_contract_630v18g = dict(
    FD004_BIAS_SAFE_CHECKPOINT_REPAIR_CONTRACT_630V18FR1
)

repair_checks_630v18g = {
    "status":
        str(
            repair_contract_630v18g.get("status")
        )
        ==
        "BIAS_CHECKPOINT_SAFE_SERIALIZATION_REPAIRED_AND_VERIFIED",

    "scientific_execution_false":
        bool(
            repair_contract_630v18g.get(
                "scientific_execution_performed"
            )
        )
        is False,

    "scientific_results_unchanged":
        bool(
            repair_contract_630v18g.get(
                "scientific_results_changed"
            )
        )
        is False,

    "v1_preserved":
        bool(
            repair_contract_630v18g.get(
                "original_v1_preserved_exact"
            )
        )
        is True,

    "v1_authenticated":
        bool(
            repair_contract_630v18g.get(
                "v1_npz_authenticated"
            )
        )
        is True
        and
        bool(
            repair_contract_630v18g.get(
                "v1_json_authenticated"
            )
        )
        is True,

    "v2_non_object":
        bool(
            repair_contract_630v18g.get(
                "v2_scenario_id_has_object_dtype"
            )
        )
        is False,

    "v2_roundtrip_all_true":
        all(
            bool(value_630v18g)
            for value_630v18g
            in dict(
                repair_contract_630v18g.get(
                    "v2_allow_pickle_false_roundtrip",
                    {},
                )
            ).values()
        ),

    "v2_json_all_true":
        all(
            bool(value_630v18g)
            for value_630v18g
            in dict(
                repair_contract_630v18g.get(
                    "v2_json_checks",
                    {},
                )
            ).values()
        ),

    "no_dev_outcomes":
        bool(
            repair_contract_630v18g.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "no_threshold":
        bool(
            repair_contract_630v18g.get(
                "threshold_selected"
            )
        )
        is False,

    "no_policy":
        bool(
            repair_contract_630v18g.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            repair_contract_630v18g.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            repair_contract_630v18g.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(repair_checks_630v18g.values()):
    failed_630v18g = [
        key_630v18g
        for key_630v18g, passed_630v18g
        in repair_checks_630v18g.items()
        if not passed_630v18g
    ]

    raise RuntimeError(
        "Bias safe-checkpoint closure check failed: "
        + ", ".join(failed_630v18g)
    )

# =============================================================================
# 3) PATH PRESENCE ONLY — NO FILE REWRITE
# =============================================================================

safe_npz_path_630v18g = Path(
    FD004_BIAS_SAFE_CHECKPOINT_NPZ_PATH_630V18FR1
)

safe_json_path_630v18g = Path(
    FD004_BIAS_SAFE_CHECKPOINT_JSON_PATH_630V18FR1
)

safe_paths_present_630v18g = bool(
    safe_npz_path_630v18g.is_file()
    and
    safe_json_path_630v18g.is_file()
)

if not safe_paths_present_630v18g:
    raise RuntimeError(
        "Verified safe bias checkpoint file(s) are missing."
    )

# =============================================================================
# 4) FINAL BIAS CLOSURE
# =============================================================================

FD004_BIAS_FINAL_CLOSURE_630V18G = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18G",

    "family":
        "bias",

    "status":
        "BIAS_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        102,

    "prediction_block_sha256":
        observed_630v18g["prediction"],

    "mc_std_block_sha256":
        observed_630v18g["mc_std"],

    "final_risk_block_sha256":
        observed_630v18g["final_risk"],

    "bias_block_contract_sha256":
        observed_630v18g["bias_block_contract"],

    "safe_checkpoint_repair_sha256":
        observed_630v18g["safe_checkpoint_repair"],

    "safe_npz_sha256":
        observed_630v18g["safe_npz"],

    "safe_json_sha256":
        observed_630v18g["safe_json"],

    "safe_checkpoint_paths_present":
        safe_paths_present_630v18g,

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G = (
    hashlib.sha256(
        json.dumps(
            FD004_BIAS_FINAL_CLOSURE_630V18G,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_BIAS_FAMILY_CLOSED_630V18G = True
FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18G = False
FD004_DEV_TARGETS_ERRORS_USED_630V18G = False
FD004_THRESHOLD_SELECTED_630V18G = False
FD004_POLICY_SELECTED_630V18G = False
FD004_TEST_CONTENTS_READ_630V18G = False
FD004_RUL_CONTENTS_READ_630V18G = False

all_checks_passed_630v18g = True

# =============================================================================
# 5) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18G — FD004 BIAS FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Bias scenarios rerun              :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FROZEN BIAS EVIDENCE ---")

for key_630v18g in (
    "prediction",
    "mc_std",
    "final_risk",
    "bias_block_contract",
    "safe_checkpoint_repair",
    "safe_npz",
    "safe_json",
):
    print(
        f"{key_630v18g:28s}:",
        hash_checks_630v18g[key_630v18g],
    )

print("\n--- SAFE CHECKPOINT CLOSURE ---")

for key_630v18g, value_630v18g in (
    repair_checks_630v18g.items()
):
    print(
        f"{key_630v18g:28s}:",
        value_630v18g,
    )

print(
    "Safe checkpoint files present    :",
    safe_paths_present_630v18g,
)

print("\n--- FINAL CLOSURE ---")

print(
    "Bias family closed               :",
    FD004_BIAS_FAMILY_CLOSED_630V18G,
)

print(
    "Bias final closure SHA-256       :",
    FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G,
)

print(
    "\nPASS — FD004 bias is now closed as evidence-locked, "
    "with the 102-scenario scientific block unchanged and the "
    "pickle-free safe checkpoint verified."
)

print(
    "Next: execute only the 51 admissible noise scenarios under "
    "the same frozen label-free pipeline."
)

In [ ]:
# STEP 6.30V-18H — FD004 NOISE FAMILY EXECUTION
# 51 admissible noise scenarios only.
#
# LABEL-FREE:
# corruption -> normalization -> windows -> deterministic prediction
# -> 50-pass MC dropout -> reliability chain
#
# NO DEV targets/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.
#
# Retain:
# - deterministic prediction
# - MC STD
# - final selective risk
# - SHA-256 of each full MC matrix, then discard full MC matrix.
#
# Checkpoint:
# - fixed-width Unicode scenario IDs
# - allow_pickle=False round-trip verification
# - persistence failure does NOT invalidate scientific execution.

import numpy as np
import pandas as pd
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18h = (
    "all_checks_passed_630v18g",
    "FD004_BIAS_FAMILY_CLOSED_630V18G",
    "FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_DETERMINISTIC_PREDICTOR_630V18CR1",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
    "FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0",
)

missing_630v18h = [
    name
    for name in required_630v18h
    if name not in globals()
]

if missing_630v18h:
    raise RuntimeError(
        "Missing STEP 6.30V-18H authority object(s): "
        + ", ".join(missing_630v18h)
    )

if not bool(all_checks_passed_630v18g):
    raise RuntimeError(
        "STEP 6.30V-18G must remain passed."
    )

if not bool(FD004_BIAS_FAMILY_CLOSED_630V18G):
    raise RuntimeError(
        "Bias family must remain evidence-locked before noise."
    )

EXPECTED_BIAS_CLOSURE_SHA256_630V18H = (
    "68dbb6cd7fde080687492fea10a53cdf305b227717b411bb507e3ef4f99bb353"
)

if (
    str(FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G)
    != EXPECTED_BIAS_CLOSURE_SHA256_630V18H
):
    raise RuntimeError(
        "Bias final-closure fingerprint changed."
    )


# =============================================================================
# 1) FREEZE NOISE BLOCK FROM EXECUTION MANIFEST
# =============================================================================

execution_manifest_630v18h = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

noise_manifest_630v18h = (
    execution_manifest_630v18h.loc[
        execution_manifest_630v18h[
            "family"
        ].astype(str)
        == "noise"
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(noise_manifest_630v18h) != 51:
    raise RuntimeError(
        "Expected exactly 51 admissible noise scenarios."
    )

noise_numbers_630v18h = (
    noise_manifest_630v18h[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

if (
    int(noise_numbers_630v18h.min()) != 130
    or
    int(noise_numbers_630v18h.max()) != 189
):
    raise RuntimeError(
        "Frozen noise scenario-number range changed."
    )

if not np.all(
    noise_manifest_630v18h[
        "family"
    ].astype(str).to_numpy()
    == "noise"
):
    raise RuntimeError(
        "Non-noise row entered the noise block."
    )

if not (
    noise_manifest_630v18h[
        "alpha"
    ].notna().all()
    and
    noise_manifest_630v18h[
        "seed"
    ].notna().all()
):
    raise RuntimeError(
        "Noise manifest has missing alpha/seed values."
    )


# =============================================================================
# 2) FROZEN EXECUTION AUTHORITIES
# =============================================================================

base_630v18h = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v18h = (
    base_630v18h.copy(deep=True)
)

regime_ids_630v18h = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

expected_window_units_630v18h = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

expected_endpoint_cycles_630v18h = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if expected_window_units_630v18h.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV window-unit authority shape changed."
    )

if expected_endpoint_cycles_630v18h.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle authority shape changed."
    )

model_630v18h = MODEL_FD004_630I
device_630v18h = FD004_DEVICE_630I

if bool(model_630v18h.training):
    raise RuntimeError(
        "FD004 model must be in eval mode before noise execution."
    )


def _model_state_sha_630v18h(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        arr = tensor.numpy()

        h.update(
            key.encode("utf-8")
        )

        h.update(
            str(arr.dtype).encode("utf-8")
        )

        h.update(
            str(tuple(arr.shape)).encode("utf-8")
        )

        h.update(
            arr.tobytes(order="C")
        )

    return h.hexdigest()


model_state_before_630v18h = (
    _model_state_sha_630v18h(
        model_630v18h
    )
)

model_mode_before_630v18h = bool(
    model_630v18h.training
)


# =============================================================================
# 3) PREALLOCATE ONLY FROZEN RETAINED PRODUCTS
# =============================================================================

n_scenarios_630v18h = 51
n_windows_630v18h = 10824

noise_pred_matrix_630v18h = np.empty(
    (n_scenarios_630v18h, n_windows_630v18h),
    dtype=np.float32,
)

noise_mc_std_matrix_630v18h = np.empty(
    (n_scenarios_630v18h, n_windows_630v18h),
    dtype=np.float64,
)

noise_final_risk_matrix_630v18h = np.empty(
    (n_scenarios_630v18h, n_windows_630v18h),
    dtype=np.float64,
)

noise_mc_matrix_hashes_630v18h = []
noise_scenario_evidence_630v18h = []


# =============================================================================
# 4) EXECUTE EXACTLY 51 ADMISSIBLE NOISE SCENARIOS
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18H — FD004 NOISE FAMILY EXECUTION "
    "(51 ADMISSIBLE SCENARIOS)"
)

print("=" * 176)

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nStarting frozen noise block        :",
    n_scenarios_630v18h,
    "scenarios",
)


for row_index_630v18h, row_630v18h in (
    noise_manifest_630v18h.iterrows()
):

    scenario_id_630v18h = str(
        row_630v18h["scenario_id"]
    )

    scenario_number_630v18h = int(
        row_630v18h["scenario_number"]
    )

    sensor_630v18h = str(
        row_630v18h["sensor"]
    )

    alpha_630v18h = float(
        row_630v18h["alpha"]
    )

    seed_630v18h = int(
        row_630v18h["seed"]
    )

    # -------------------------------------------------------------------------
    # A) Frozen corruption dispatcher
    # -------------------------------------------------------------------------

    corrupted_raw_630v18h = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v18h,
            row_630v18h,
        )
    )

    if not isinstance(
        corrupted_raw_630v18h,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: dispatcher did not return DataFrame."
        )

    if corrupted_raw_630v18h.shape != (12274, 27):
        raise RuntimeError(
            f"{scenario_id_630v18h}: corrupted raw shape changed."
        )

    if not np.array_equal(
        corrupted_raw_630v18h[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_ids_630v18h,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: regime_id changed under noise."
        )

    # -------------------------------------------------------------------------
    # B) Frozen normalization
    # -------------------------------------------------------------------------

    normalized_630v18h = (
        normalize_fd004_raw_630v2(
            corrupted_raw_630v18h,
            regime_ids_630v18h,
        )
    )

    # -------------------------------------------------------------------------
    # C) Frozen windows
    # -------------------------------------------------------------------------

    (
        X_630v18h,
        unit_ids_630v18h,
        endpoint_cycles_630v18h,
    ) = build_windows_fd004_630v2(
        normalized_630v18h
    )

    X_630v18h = np.asarray(
        X_630v18h,
        dtype=np.float32,
    )

    if X_630v18h.shape != (10824, 30, 24):
        raise RuntimeError(
            f"{scenario_id_630v18h}: window tensor shape changed."
        )

    if not np.isfinite(
        X_630v18h
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18h}: non-finite model input."
        )

    if not np.array_equal(
        np.asarray(
            unit_ids_630v18h,
            dtype=np.int64,
        ).reshape(-1),
        expected_window_units_630v18h,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: window-unit membership changed."
        )

    if not np.array_equal(
        np.asarray(
            endpoint_cycles_630v18h,
            dtype=np.int64,
        ).reshape(-1),
        expected_endpoint_cycles_630v18h,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: endpoint-cycle alignment changed."
        )

    # -------------------------------------------------------------------------
    # D) Frozen deterministic inference
    # -------------------------------------------------------------------------

    deterministic_pred_630v18h = np.asarray(
        FD004_DETERMINISTIC_PREDICTOR_630V18CR1(
            X_630v18h,
            model_630v18h,
            device_630v18h,
            batch_size=512,
        ),
        dtype=np.float32,
    ).reshape(-1)

    if deterministic_pred_630v18h.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18h}: deterministic prediction shape changed."
        )

    if not np.isfinite(
        deterministic_pred_630v18h
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18h}: non-finite deterministic prediction."
        )

    # -------------------------------------------------------------------------
    # E) Frozen 50-pass MC
    # -------------------------------------------------------------------------

    (
        mc_matrix_630v18h,
        mc_mean_630v18h,
        mc_std_630v18h,
    ) = fd004_mc_dropout_predict_630v4r2(
        X_630v18h,
        model_630v18h,
        device_630v18h,
    )

    mc_matrix_630v18h = np.asarray(
        mc_matrix_630v18h,
        dtype=np.float32,
    )

    mc_std_630v18h = np.asarray(
        mc_std_630v18h,
        dtype=np.float64,
    ).reshape(-1)

    if mc_matrix_630v18h.shape != (50, 10824):
        raise RuntimeError(
            f"{scenario_id_630v18h}: MC matrix shape changed."
        )

    if mc_std_630v18h.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18h}: MC STD shape changed."
        )

    if not (
        np.isfinite(mc_matrix_630v18h).all()
        and
        np.isfinite(mc_std_630v18h).all()
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: non-finite MC output."
        )

    if np.any(
        mc_std_630v18h < 0.0
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: negative MC STD."
        )

    mc_matrix_sha_630v18h = str(
        sha256_array_630v2(
            mc_matrix_630v18h
        )
    )

    # -------------------------------------------------------------------------
    # F) Frozen reliability chain
    # -------------------------------------------------------------------------

    (
        phase_ids_630v18h,
        adjusted_630v18h,
        g1_630v18h,
        final_risk_630v18h,
    ) = apply_fd004_final_reliability_chain_630v4r2(
        deterministic_pred_630v18h,
        mc_std_630v18h,
    )

    final_risk_630v18h = np.asarray(
        final_risk_630v18h,
        dtype=np.float64,
    ).reshape(-1)

    if final_risk_630v18h.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18h}: final risk shape changed."
        )

    if not np.isfinite(
        final_risk_630v18h
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18h}: non-finite final risk."
        )

    if np.any(
        (final_risk_630v18h < 0.0)
        |
        (final_risk_630v18h > 1.0)
    ):
        raise RuntimeError(
            f"{scenario_id_630v18h}: final risk escaped [0,1]."
        )

    # -------------------------------------------------------------------------
    # G) Retain frozen required products
    # -------------------------------------------------------------------------

    noise_pred_matrix_630v18h[
        row_index_630v18h
    ] = deterministic_pred_630v18h

    noise_mc_std_matrix_630v18h[
        row_index_630v18h
    ] = mc_std_630v18h

    noise_final_risk_matrix_630v18h[
        row_index_630v18h
    ] = final_risk_630v18h

    noise_mc_matrix_hashes_630v18h.append(
        mc_matrix_sha_630v18h
    )

    noise_scenario_evidence_630v18h.append(
        {
            "scenario_id":
                scenario_id_630v18h,

            "scenario_number":
                scenario_number_630v18h,

            "family":
                "noise",

            "sensor":
                sensor_630v18h,

            "alpha":
                alpha_630v18h,

            "seed":
                seed_630v18h,

            "deterministic_prediction_sha256":
                str(
                    sha256_array_630v2(
                        deterministic_pred_630v18h
                    )
                ),

            "mc_matrix_sha256":
                mc_matrix_sha_630v18h,

            "mc_std_sha256":
                str(
                    sha256_array_630v2(
                        mc_std_630v18h
                    )
                ),

            "final_selective_risk_sha256":
                str(
                    sha256_array_630v2(
                        final_risk_630v18h
                    )
                ),
        }
    )

    # Discard products not retained by frozen contract.
    del mc_matrix_630v18h
    del mc_mean_630v18h
    del phase_ids_630v18h
    del adjusted_630v18h
    del g1_630v18h

    del corrupted_raw_630v18h
    del normalized_630v18h
    del X_630v18h

    if (
        (row_index_630v18h + 1) == 1
        or
        (row_index_630v18h + 1) % 10 == 0
        or
        (row_index_630v18h + 1) == n_scenarios_630v18h
    ):
        print(
            f"Completed noise scenario "
            f"{row_index_630v18h + 1:2d}/"
            f"{n_scenarios_630v18h}: "
            f"{scenario_id_630v18h}"
        )


# =============================================================================
# 5) POST-BLOCK SCIENTIFIC SAFETY
# =============================================================================

if not np.isfinite(
    noise_pred_matrix_630v18h
).all():
    raise RuntimeError(
        "Retained noise deterministic predictions contain non-finite values."
    )

if not np.isfinite(
    noise_mc_std_matrix_630v18h
).all():
    raise RuntimeError(
        "Retained noise MC STD contains non-finite values."
    )

if not np.isfinite(
    noise_final_risk_matrix_630v18h
).all():
    raise RuntimeError(
        "Retained noise final risk contains non-finite values."
    )

if np.any(
    noise_mc_std_matrix_630v18h < 0.0
):
    raise RuntimeError(
        "Retained noise MC STD contains negative values."
    )

if np.any(
    (noise_final_risk_matrix_630v18h < 0.0)
    |
    (noise_final_risk_matrix_630v18h > 1.0)
):
    raise RuntimeError(
        "Retained noise risk escaped [0,1]."
    )

if len(
    noise_mc_matrix_hashes_630v18h
) != 51:
    raise RuntimeError(
        "Noise MC matrix hash count changed."
    )

if len(
    noise_scenario_evidence_630v18h
) != 51:
    raise RuntimeError(
        "Noise scenario evidence count changed."
    )

base_unchanged_630v18h = bool(
    base_630v18h.equals(
        base_snapshot_630v18h
    )
)

if not base_unchanged_630v18h:
    raise RuntimeError(
        "Frozen corruption base changed during noise execution."
    )

model_state_after_630v18h = (
    _model_state_sha_630v18h(
        model_630v18h
    )
)

model_mode_after_630v18h = bool(
    model_630v18h.training
)

model_state_unchanged_630v18h = bool(
    model_state_before_630v18h
    ==
    model_state_after_630v18h
)

model_mode_unchanged_630v18h = bool(
    model_mode_before_630v18h
    ==
    model_mode_after_630v18h
)

if not model_state_unchanged_630v18h:
    raise RuntimeError(
        "Model state changed during noise block."
    )

if not model_mode_unchanged_630v18h:
    raise RuntimeError(
        "Model mode changed during noise block."
    )


# =============================================================================
# 6) FREEZE SCIENTIFIC NOISE EVIDENCE
# =============================================================================

FD004_NOISE_SCENARIO_MANIFEST_630V18H = (
    noise_manifest_630v18h.copy()
)

FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H = (
    noise_pred_matrix_630v18h
)

FD004_NOISE_MC_STD_630V18H = (
    noise_mc_std_matrix_630v18h
)

FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H = (
    noise_final_risk_matrix_630v18h
)

FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H = tuple(
    noise_mc_matrix_hashes_630v18h
)

FD004_NOISE_SCENARIO_EVIDENCE_630V18H = tuple(
    noise_scenario_evidence_630v18h
)

FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H = str(
    sha256_array_630v2(
        FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H
    )
)

FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H = str(
    sha256_array_630v2(
        FD004_NOISE_MC_STD_630V18H
    )
)

FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H = str(
    sha256_array_630v2(
        FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H
    )
)


noise_block_contract_630v18h = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18H",

    "family":
        "noise",

    "status":
        "NOISE_BLOCK_COMPLETE_LABEL_FREE",

    "scenario_count":
        51,

    "scenario_number_min":
        int(noise_numbers_630v18h.min()),

    "scenario_number_max":
        int(noise_numbers_630v18h.max()),

    "deterministic_prediction_shape":
        tuple(
            FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H.shape
        ),

    "mc_std_shape":
        tuple(
            FD004_NOISE_MC_STD_630V18H.shape
        ),

    "final_selective_risk_shape":
        tuple(
            FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H.shape
        ),

    "full_mc_matrix_hash_count":
        len(
            FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H
        ),

    "full_mc_matrices_retained":
        False,

    "prediction_block_sha256":
        FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H,

    "mc_std_block_sha256":
        FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H,

    "final_risk_block_sha256":
        FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H,

    "bias_final_closure_sha256":
        str(
            FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G
        ),

    "admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helper_return_audit_sha256":
        str(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0
        ),

    "corruption_base_unchanged":
        base_unchanged_630v18h,

    "model_state_unchanged":
        model_state_unchanged_630v18h,

    "model_mode_unchanged":
        model_mode_unchanged_630v18h,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_NOISE_BLOCK_CONTRACT_630V18H = (
    noise_block_contract_630v18h
)

FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H = (
    hashlib.sha256(
        json.dumps(
            noise_block_contract_630v18h,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) SAFE CHECKPOINT — FIXED-WIDTH UNICODE FROM THE START
# =============================================================================

checkpoint_saved_630v18h = False
checkpoint_roundtrip_exact_630v18h = False
checkpoint_error_630v18h = None

checkpoint_npz_path_630v18h = None
checkpoint_json_path_630v18h = None

checkpoint_npz_sha_630v18h = None
checkpoint_json_sha_630v18h = None

checkpoint_roundtrip_checks_630v18h = None

try:
    checkpoint_dir_630v18h = Path(
        r"C:\RUL_Reliability_Research\checkpoints"
    )

    checkpoint_dir_630v18h.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_npz_path_630v18h = (
        checkpoint_dir_630v18h
        /
        "fd004_noise_block_630v18h_v1_safe.npz"
    )

    checkpoint_json_path_630v18h = (
        checkpoint_dir_630v18h
        /
        "fd004_noise_block_630v18h_v1_safe.json"
    )

    npz_tmp_630v18h = (
        checkpoint_dir_630v18h
        /
        "fd004_noise_block_630v18h_v1_safe.npz.tmp"
    )

    json_tmp_630v18h = (
        checkpoint_dir_630v18h
        /
        "fd004_noise_block_630v18h_v1_safe.json.tmp"
    )

    scenario_id_list_630v18h = (
        noise_manifest_630v18h[
            "scenario_id"
        ]
        .astype(str)
        .tolist()
    )

    scenario_id_max_len_630v18h = max(
        len(value)
        for value in scenario_id_list_630v18h
    )

    scenario_id_unicode_630v18h = np.asarray(
        scenario_id_list_630v18h,
        dtype=f"<U{scenario_id_max_len_630v18h}",
    )

    full_mc_hash_unicode_630v18h = np.asarray(
        FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H,
        dtype="<U64",
    )

    with open(
        npz_tmp_630v18h,
        "wb",
    ) as file:
        np.savez_compressed(
            file,

            scenario_number=
                noise_manifest_630v18h[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),

            scenario_id=
                scenario_id_unicode_630v18h,

            deterministic_prediction=
                FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,

            mc_std=
                FD004_NOISE_MC_STD_630V18H,

            final_selective_risk=
                FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H,

            full_mc_matrix_sha256=
                full_mc_hash_unicode_630v18h,
        )

    os.replace(
        npz_tmp_630v18h,
        checkpoint_npz_path_630v18h,
    )

    def _file_sha256_630v18h(path):
        h = hashlib.sha256()

        with open(path, "rb") as file:
            while True:
                chunk = file.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                h.update(chunk)

        return h.hexdigest()

    checkpoint_npz_sha_630v18h = (
        _file_sha256_630v18h(
            checkpoint_npz_path_630v18h
        )
    )

    checkpoint_metadata_630v18h = {
        "dataset":
            "FD004",

        "stage":
            "6.30V-18H",

        "family":
            "noise",

        "status":
            "SAFE_SERIALIZATION_CHECKPOINT",

        "scenario_count":
            51,

        "scenario_id_dtype":
            str(
                scenario_id_unicode_630v18h.dtype
            ),

        "block_contract_sha256":
            FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H,

        "prediction_block_sha256":
            FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H,

        "mc_std_block_sha256":
            FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H,

        "final_risk_block_sha256":
            FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H,

        "npz_sha256":
            checkpoint_npz_sha_630v18h,

        "scenario_evidence":
            list(
                FD004_NOISE_SCENARIO_EVIDENCE_630V18H
            ),

        "dev_targets_errors_used":
            False,

        "threshold_selected":
            False,

        "policy_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    }

    with open(
        json_tmp_630v18h,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            checkpoint_metadata_630v18h,
            file,
            sort_keys=True,
            indent=2,
            default=str,
        )

    os.replace(
        json_tmp_630v18h,
        checkpoint_json_path_630v18h,
    )

    checkpoint_json_sha_630v18h = (
        _file_sha256_630v18h(
            checkpoint_json_path_630v18h
        )
    )

    # -------------------------------------------------------------------------
    # Immediate safe round-trip with allow_pickle=False
    # -------------------------------------------------------------------------

    with np.load(
        checkpoint_npz_path_630v18h,
        allow_pickle=False,
    ) as checkpoint_npz_630v18h:

        loaded_numbers_630v18h = np.asarray(
            checkpoint_npz_630v18h[
                "scenario_number"
            ],
            dtype=np.int64,
        ).reshape(-1)

        loaded_id_raw_630v18h = np.asarray(
            checkpoint_npz_630v18h[
                "scenario_id"
            ]
        )

        loaded_ids_630v18h = (
            loaded_id_raw_630v18h
            .astype(str)
            .reshape(-1)
        )

        loaded_pred_630v18h = np.asarray(
            checkpoint_npz_630v18h[
                "deterministic_prediction"
            ],
            dtype=np.float32,
        )

        loaded_std_630v18h = np.asarray(
            checkpoint_npz_630v18h[
                "mc_std"
            ],
            dtype=np.float64,
        )

        loaded_risk_630v18h = np.asarray(
            checkpoint_npz_630v18h[
                "final_selective_risk"
            ],
            dtype=np.float64,
        )

        loaded_mc_hashes_630v18h = tuple(
            str(x)
            for x in np.asarray(
                checkpoint_npz_630v18h[
                    "full_mc_matrix_sha256"
                ]
            ).reshape(-1)
        )

    checkpoint_roundtrip_checks_630v18h = {
        "scenario_id_non_object":
            bool(
                not loaded_id_raw_630v18h.dtype.hasobject
            ),

        "scenario_number":
            bool(
                np.array_equal(
                    loaded_numbers_630v18h,
                    noise_manifest_630v18h[
                        "scenario_number"
                    ].to_numpy(dtype=np.int64),
                )
            ),

        "scenario_id":
            bool(
                np.array_equal(
                    loaded_ids_630v18h,
                    scenario_id_unicode_630v18h,
                )
            ),

        "prediction":
            bool(
                np.array_equal(
                    loaded_pred_630v18h,
                    FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,
                )
            ),

        "mc_std":
            bool(
                np.array_equal(
                    loaded_std_630v18h,
                    FD004_NOISE_MC_STD_630V18H,
                )
            ),

        "final_risk":
            bool(
                np.array_equal(
                    loaded_risk_630v18h,
                    FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H,
                )
            ),

        "full_mc_hashes":
            loaded_mc_hashes_630v18h
            ==
            tuple(
                FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H
            ),
    }

    checkpoint_roundtrip_exact_630v18h = bool(
        all(
            checkpoint_roundtrip_checks_630v18h.values()
        )
    )

    if not checkpoint_roundtrip_exact_630v18h:
        raise RuntimeError(
            "Noise safe checkpoint round-trip verification failed."
        )

    checkpoint_saved_630v18h = True

except Exception as exc_630v18h:
    checkpoint_error_630v18h = repr(
        exc_630v18h
    )


# =============================================================================
# 8) FREEZE CHECKPOINT STATUS
# =============================================================================

FD004_NOISE_CHECKPOINT_SAVED_630V18H = bool(
    checkpoint_saved_630v18h
)

FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H = bool(
    checkpoint_roundtrip_exact_630v18h
)

FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H = (
    None
    if checkpoint_npz_path_630v18h is None
    else str(checkpoint_npz_path_630v18h)
)

FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H = (
    None
    if checkpoint_json_path_630v18h is None
    else str(checkpoint_json_path_630v18h)
)

FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H = (
    checkpoint_npz_sha_630v18h
)

FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H = (
    checkpoint_json_sha_630v18h
)

FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H = (
    checkpoint_roundtrip_checks_630v18h
)

FD004_NOISE_CHECKPOINT_ERROR_630V18H = (
    checkpoint_error_630v18h
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD004_NOISE_BLOCK_COMPLETE_630V18H = True
FD004_NOISE_SCENARIOS_EXECUTED_630V18H = 51

FD004_DEV_TARGETS_ERRORS_USED_630V18H = False
FD004_THRESHOLD_SELECTED_630V18H = False
FD004_POLICY_SELECTED_630V18H = False
FD004_TEST_CONTENTS_READ_630V18H = False
FD004_RUL_CONTENTS_READ_630V18H = False

all_checks_passed_630v18h = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("\n--- NOISE BLOCK COMPLETE ---")

print(
    "Noise scenarios executed          :",
    FD004_NOISE_SCENARIOS_EXECUTED_630V18H,
)

print(
    "Deterministic retained shape      :",
    FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H.shape,
)

print(
    "MC STD retained shape             :",
    FD004_NOISE_MC_STD_630V18H.shape,
)

print(
    "Final risk retained shape         :",
    FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H.shape,
)

print(
    "Full MC matrix hashes retained    :",
    len(
        FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H
    ),
)

print(
    "Full MC matrices retained         :",
    False,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18h,
)

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18h,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18h,
)

print(
    "Prediction block SHA-256          :",
    FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H,
)

print(
    "MC STD block SHA-256              :",
    FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H,
)

print(
    "Final risk block SHA-256          :",
    FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H,
)

print(
    "Noise block contract SHA-256      :",
    FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H,
)


print("\n--- LABEL-FREE BLOCK SUMMARY ---")

print(
    "MC STD min / median / mean / max  :",
    (
        float(
            np.min(
                FD004_NOISE_MC_STD_630V18H
            )
        ),
        float(
            np.median(
                FD004_NOISE_MC_STD_630V18H
            )
        ),
        float(
            np.mean(
                FD004_NOISE_MC_STD_630V18H
            )
        ),
        float(
            np.max(
                FD004_NOISE_MC_STD_630V18H
            )
        ),
    ),
)

print(
    "Risk min / median / mean / max    :",
    (
        float(
            np.min(
                FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H
            )
        ),
        float(
            np.median(
                FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H
            )
        ),
        float(
            np.mean(
                FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H
            )
        ),
        float(
            np.max(
                FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H
            )
        ),
    ),
)


print("\n--- SAFE CHECKPOINT ---")

print(
    "Checkpoint saved                  :",
    FD004_NOISE_CHECKPOINT_SAVED_630V18H,
)

print(
    "allow_pickle=False round-trip     :",
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H,
)

print(
    "NPZ path                          :",
    FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H,
)

print(
    "JSON path                         :",
    FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H,
)

print(
    "NPZ SHA-256                       :",
    FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H,
)

print(
    "JSON SHA-256                      :",
    FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H,
)

print(
    "Round-trip checks                 :",
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H,
)

print(
    "Checkpoint error                  :",
    FD004_NOISE_CHECKPOINT_ERROR_630V18H,
)


print("\n--- LEAKAGE / POLICY GUARDS ---")

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nPASS — the 51 admissible FD004 noise scenarios are complete "
    "under the frozen label-free execution contract."
)

if (
    checkpoint_saved_630v18h
    and checkpoint_roundtrip_exact_630v18h
):
    print(
        "The noise block also has a verified pickle-free checkpoint. "
        "Next: close noise evidence, then proceed to drift."
    )
else:
    print(
        "The scientific noise block is complete, but checkpoint "
        "persistence/verification failed. Do NOT rerun noise; "
        "repair persistence only."
    )

In [ ]:
# STEP 6.30V-18I — FD004 NOISE FAMILY FINAL EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.

import numpy as np
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18i = (
    "all_checks_passed_630v18h",
    "FD004_NOISE_BLOCK_COMPLETE_630V18H",
    "FD004_NOISE_SCENARIOS_EXECUTED_630V18H",
    "FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H",
    "FD004_NOISE_MC_STD_630V18H",
    "FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H",
    "FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H",
    "FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H",
    "FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H",
    "FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H",
    "FD004_NOISE_BLOCK_CONTRACT_630V18H",
    "FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H",
    "FD004_NOISE_CHECKPOINT_SAVED_630V18H",
    "FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H",
    "FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H",
    "FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H",
    "FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H",
    "FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H",
    "FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H",
    "sha256_array_630v2",
)

missing_630v18i = [
    name
    for name in required_630v18i
    if name not in globals()
]

if missing_630v18i:
    raise RuntimeError(
        "Missing STEP 6.30V-18I authority object(s): "
        + ", ".join(missing_630v18i)
    )

if not bool(all_checks_passed_630v18h):
    raise RuntimeError(
        "STEP 6.30V-18H must remain passed."
    )

if not bool(FD004_NOISE_BLOCK_COMPLETE_630V18H):
    raise RuntimeError(
        "Noise block is not marked complete."
    )

if int(FD004_NOISE_SCENARIOS_EXECUTED_630V18H) != 51:
    raise RuntimeError(
        "Noise executed-scenario count changed."
    )


# =============================================================================
# 1) EXACT FROZEN 18H HASH GUARDS
# =============================================================================

expected_hashes_630v18i = {
    "prediction":
        "e407372bb546ec43ecd504dc2a50a5e4b59aef0786948cd5174a301d8fef5517",

    "mc_std":
        "172fe25f660c664ffce10bf9d0a614744dff300d00c34859746a949fd724eac5",

    "final_risk":
        "c1dac03533345fef947d873225f5667363955e41e3c07cad87735602e85a4010",

    "block_contract":
        "c760196d352fa49dd33aac40f00dd0683ae9fe21c1ea2d84518cfcad4aa70b60",

    "checkpoint_npz":
        "cee6aba869cec8680fe8f4557b10cfd729ef0f5e62637f070ddcfb3895e55b9c",

    "checkpoint_json":
        "cae0dd5dc606d5772e159f6052584dd9744c5ad847dee3a36d55a9fb939a0e666",
}

observed_hashes_630v18i = {
    "prediction":
        str(
            FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H
        ),

    "mc_std":
        str(
            FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H
        ),

    "final_risk":
        str(
            FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H
        ),

    "block_contract":
        str(
            FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H
        ),

    "checkpoint_npz":
        str(
            FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H
        ),

    "checkpoint_json":
        str(
            FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H
        ),
}

hash_guards_630v18i = {
    key:
        observed_hashes_630v18i[key]
        ==
        expected_hashes_630v18i[key]
    for key in expected_hashes_630v18i
}

if not all(hash_guards_630v18i.values()):
    failed_630v18i = [
        key
        for key, passed
        in hash_guards_630v18i.items()
        if not passed
    ]

    raise RuntimeError(
        "Frozen noise evidence hash changed: "
        + ", ".join(failed_630v18i)
    )


# =============================================================================
# 2) RESIDENT SCIENTIFIC ARRAYS REHASH
# =============================================================================

pred_630v18i = np.asarray(
    FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,
    dtype=np.float32,
)

std_630v18i = np.asarray(
    FD004_NOISE_MC_STD_630V18H,
    dtype=np.float64,
)

risk_630v18i = np.asarray(
    FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H,
    dtype=np.float64,
)

shape_checks_630v18i = {
    "prediction":
        pred_630v18i.shape == (51, 10824),

    "mc_std":
        std_630v18i.shape == (51, 10824),

    "final_risk":
        risk_630v18i.shape == (51, 10824),
}

if not all(shape_checks_630v18i.values()):
    raise RuntimeError(
        "Noise retained-array shape changed."
    )

validity_checks_630v18i = {
    "prediction_finite":
        bool(np.isfinite(pred_630v18i).all()),

    "mc_std_finite":
        bool(np.isfinite(std_630v18i).all()),

    "mc_std_nonnegative":
        bool(np.all(std_630v18i >= 0.0)),

    "risk_finite":
        bool(np.isfinite(risk_630v18i).all()),

    "risk_in_unit_interval":
        bool(
            np.all(
                (risk_630v18i >= 0.0)
                &
                (risk_630v18i <= 1.0)
            )
        ),
}

if not all(validity_checks_630v18i.values()):
    raise RuntimeError(
        "Noise retained-array validity check failed."
    )

rehash_checks_630v18i = {
    "prediction":
        str(
            sha256_array_630v2(
                pred_630v18i
            )
        )
        ==
        expected_hashes_630v18i[
            "prediction"
        ],

    "mc_std":
        str(
            sha256_array_630v2(
                std_630v18i
            )
        )
        ==
        expected_hashes_630v18i[
            "mc_std"
        ],

    "final_risk":
        str(
            sha256_array_630v2(
                risk_630v18i
            )
        )
        ==
        expected_hashes_630v18i[
            "final_risk"
        ],
}

if not all(rehash_checks_630v18i.values()):
    raise RuntimeError(
        "Resident noise scientific-array rehash failed."
    )


# =============================================================================
# 3) FULL-MC HASH EVIDENCE
# =============================================================================

mc_hashes_630v18i = tuple(
    str(value)
    for value
    in FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H
)

mc_hash_checks_630v18i = {
    "count_51":
        len(mc_hashes_630v18i) == 51,

    "all_sha256_format":
        bool(
            all(
                len(value) == 64
                and
                all(
                    char in "0123456789abcdef"
                    for char in value.lower()
                )
                for value in mc_hashes_630v18i
            )
        ),
}

if not all(mc_hash_checks_630v18i.values()):
    raise RuntimeError(
        "Noise full-MC hash evidence changed."
    )


# =============================================================================
# 4) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18i = dict(
    FD004_NOISE_BLOCK_CONTRACT_630V18H
)

recomputed_contract_sha_630v18i = (
    hashlib.sha256(
        json.dumps(
            contract_630v18i,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

contract_checks_630v18i = {
    "sha_exact":
        recomputed_contract_sha_630v18i
        ==
        expected_hashes_630v18i[
            "block_contract"
        ],

    "family":
        str(
            contract_630v18i.get(
                "family"
            )
        )
        ==
        "noise",

    "scenario_count":
        int(
            contract_630v18i.get(
                "scenario_count"
            )
        )
        ==
        51,

    "prediction_hash":
        str(
            contract_630v18i.get(
                "prediction_block_sha256"
            )
        )
        ==
        expected_hashes_630v18i[
            "prediction"
        ],

    "mc_std_hash":
        str(
            contract_630v18i.get(
                "mc_std_block_sha256"
            )
        )
        ==
        expected_hashes_630v18i[
            "mc_std"
        ],

    "risk_hash":
        str(
            contract_630v18i.get(
                "final_risk_block_sha256"
            )
        )
        ==
        expected_hashes_630v18i[
            "final_risk"
        ],

    "full_mc_not_retained":
        bool(
            contract_630v18i.get(
                "full_mc_matrices_retained"
            )
        )
        is False,

    "base_unchanged":
        bool(
            contract_630v18i.get(
                "corruption_base_unchanged"
            )
        )
        is True,

    "model_state_unchanged":
        bool(
            contract_630v18i.get(
                "model_state_unchanged"
            )
        )
        is True,

    "model_mode_unchanged":
        bool(
            contract_630v18i.get(
                "model_mode_unchanged"
            )
        )
        is True,

    "dev_outcomes_unused":
        bool(
            contract_630v18i.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18i.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            contract_630v18i.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18i.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18i.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(contract_checks_630v18i.values()):
    failed_630v18i = [
        key
        for key, passed
        in contract_checks_630v18i.items()
        if not passed
    ]

    raise RuntimeError(
        "Noise block contract closure failed: "
        + ", ".join(failed_630v18i)
    )


# =============================================================================
# 5) SAFE CHECKPOINT STATUS + BYTE HASHES
# =============================================================================

if not bool(
    FD004_NOISE_CHECKPOINT_SAVED_630V18H
):
    raise RuntimeError(
        "Noise checkpoint is not marked saved."
    )

if not bool(
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H
):
    raise RuntimeError(
        "Noise safe checkpoint round-trip is not marked exact."
    )

roundtrip_checks_630v18i = dict(
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H
)

if not roundtrip_checks_630v18i:
    raise RuntimeError(
        "Noise checkpoint round-trip evidence is empty."
    )

if not all(
    bool(value)
    for value
    in roundtrip_checks_630v18i.values()
):
    raise RuntimeError(
        "One or more noise checkpoint round-trip checks are false."
    )

npz_path_630v18i = Path(
    FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H
)

json_path_630v18i = Path(
    FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H
)

if not npz_path_630v18i.is_file():
    raise RuntimeError(
        "Noise safe NPZ checkpoint is missing."
    )

if not json_path_630v18i.is_file():
    raise RuntimeError(
        "Noise safe JSON checkpoint is missing."
    )


def _file_sha256_630v18i(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


checkpoint_byte_checks_630v18i = {
    "npz":
        _file_sha256_630v18i(
            npz_path_630v18i
        )
        ==
        expected_hashes_630v18i[
            "checkpoint_npz"
        ],

    "json":
        _file_sha256_630v18i(
            json_path_630v18i
        )
        ==
        expected_hashes_630v18i[
            "checkpoint_json"
        ],
}

if not all(
    checkpoint_byte_checks_630v18i.values()
):
    raise RuntimeError(
        "Noise checkpoint byte hash changed."
    )


# =============================================================================
# 6) FINAL NOISE CLOSURE
# =============================================================================

FD004_NOISE_FINAL_CLOSURE_630V18I = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18I",

    "family":
        "noise",

    "status":
        "NOISE_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        51,

    "prediction_block_sha256":
        expected_hashes_630v18i[
            "prediction"
        ],

    "mc_std_block_sha256":
        expected_hashes_630v18i[
            "mc_std"
        ],

    "final_risk_block_sha256":
        expected_hashes_630v18i[
            "final_risk"
        ],

    "noise_block_contract_sha256":
        expected_hashes_630v18i[
            "block_contract"
        ],

    "safe_checkpoint_npz_sha256":
        expected_hashes_630v18i[
            "checkpoint_npz"
        ],

    "safe_checkpoint_json_sha256":
        expected_hashes_630v18i[
            "checkpoint_json"
        ],

    "safe_checkpoint_roundtrip_exact":
        True,

    "full_mc_matrix_hash_count":
        51,

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_NOISE_FINAL_CLOSURE_SHA256_630V18I = (
    hashlib.sha256(
        json.dumps(
            FD004_NOISE_FINAL_CLOSURE_630V18I,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_NOISE_FAMILY_CLOSED_630V18I = True
FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18I = False
FD004_DEV_TARGETS_ERRORS_USED_630V18I = False
FD004_THRESHOLD_SELECTED_630V18I = False
FD004_POLICY_SELECTED_630V18I = False
FD004_TEST_CONTENTS_READ_630V18I = False
FD004_RUL_CONTENTS_READ_630V18I = False

all_checks_passed_630v18i = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18I — FD004 NOISE FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Noise scenarios rerun             :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FROZEN NOISE EVIDENCE ---")

for key, value in hash_guards_630v18i.items():
    print(
        f"{key:28s}:",
        value,
    )

print("\n--- RESIDENT SCIENTIFIC EVIDENCE ---")

for key, value in shape_checks_630v18i.items():
    print(
        f"shape_{key:22s}:",
        value,
    )

for key, value in rehash_checks_630v18i.items():
    print(
        f"rehash_{key:21s}:",
        value,
    )

print(
    "Full-MC hash count exact         :",
    mc_hash_checks_630v18i[
        "count_51"
    ],
)

print(
    "Full-MC SHA format exact         :",
    mc_hash_checks_630v18i[
        "all_sha256_format"
    ],
)

print("\n--- BLOCK CONTRACT ---")

for key, value in contract_checks_630v18i.items():
    print(
        f"{key:28s}:",
        value,
    )

print("\n--- SAFE CHECKPOINT ---")

print(
    "Round-trip exact                 :",
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H,
)

print(
    "All round-trip checks true       :",
    all(
        bool(value)
        for value
        in roundtrip_checks_630v18i.values()
    ),
)

print(
    "NPZ byte SHA exact               :",
    checkpoint_byte_checks_630v18i[
        "npz"
    ],
)

print(
    "JSON byte SHA exact              :",
    checkpoint_byte_checks_630v18i[
        "json"
    ],
)

print("\n--- FINAL CLOSURE ---")

print(
    "Noise family closed              :",
    FD004_NOISE_FAMILY_CLOSED_630V18I,
)

print(
    "Noise final closure SHA-256      :",
    FD004_NOISE_FINAL_CLOSURE_SHA256_630V18I,
)

print(
    "\nPASS — FD004 noise is now closed as evidence-locked. "
    "No noise scenario was rerun and the safe checkpoint remains exact."
)

print(
    "Next: execute only the 102 admissible drift scenarios under "
    "the same verified label-free pipeline."
)

In [ ]:
# STEP 6.30V-18I-R1 — FD004 NOISE FAMILY FINAL EVIDENCE CLOSURE
#
# Corrected version:
# NO manually transcribed scientific/checkpoint hashes.
#
# ZERO SCIENTIFIC EXECUTION.
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.

import numpy as np
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18ir1 = (
    "all_checks_passed_630v18h",
    "FD004_NOISE_BLOCK_COMPLETE_630V18H",
    "FD004_NOISE_SCENARIOS_EXECUTED_630V18H",
    "FD004_NOISE_SCENARIO_MANIFEST_630V18H",
    "FD004_NOISE_SCENARIO_EVIDENCE_630V18H",
    "FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H",
    "FD004_NOISE_MC_STD_630V18H",
    "FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H",
    "FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H",
    "FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H",
    "FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H",
    "FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H",
    "FD004_NOISE_BLOCK_CONTRACT_630V18H",
    "FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H",
    "FD004_NOISE_CHECKPOINT_SAVED_630V18H",
    "FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H",
    "FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H",
    "FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H",
    "FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H",
    "FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H",
    "FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H",
    "FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G",
    "sha256_array_630v2",
)

missing_630v18ir1 = [
    name
    for name in required_630v18ir1
    if name not in globals()
]

if missing_630v18ir1:
    raise RuntimeError(
        "Missing STEP 6.30V-18I-R1 authority object(s): "
        + ", ".join(missing_630v18ir1)
    )

if not bool(all_checks_passed_630v18h):
    raise RuntimeError(
        "STEP 6.30V-18H must remain passed."
    )

if not bool(FD004_NOISE_BLOCK_COMPLETE_630V18H):
    raise RuntimeError(
        "Noise block is not marked complete."
    )

if int(FD004_NOISE_SCENARIOS_EXECUTED_630V18H) != 51:
    raise RuntimeError(
        "Noise executed-scenario count changed."
    )

if not bool(FD004_NOISE_CHECKPOINT_SAVED_630V18H):
    raise RuntimeError(
        "Noise checkpoint is not marked saved."
    )

if not bool(FD004_NOISE_CHECKPOINT_ROUNDTRIP_EXACT_630V18H):
    raise RuntimeError(
        "Noise checkpoint is not marked round-trip exact."
    )


# =============================================================================
# 1) RESIDENT SCIENTIFIC EVIDENCE
# =============================================================================

manifest_630v18ir1 = (
    FD004_NOISE_SCENARIO_MANIFEST_630V18H
    .copy()
    .reset_index(drop=True)
)

pred_630v18ir1 = np.asarray(
    FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,
    dtype=np.float32,
)

std_630v18ir1 = np.asarray(
    FD004_NOISE_MC_STD_630V18H,
    dtype=np.float64,
)

risk_630v18ir1 = np.asarray(
    FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H,
    dtype=np.float64,
)

mc_hashes_630v18ir1 = tuple(
    str(x)
    for x in FD004_NOISE_FULL_MC_MATRIX_SHA256_630V18H
)

scenario_evidence_630v18ir1 = tuple(
    FD004_NOISE_SCENARIO_EVIDENCE_630V18H
)


shape_checks_630v18ir1 = {
    "prediction":
        pred_630v18ir1.shape == (51, 10824),

    "mc_std":
        std_630v18ir1.shape == (51, 10824),

    "final_risk":
        risk_630v18ir1.shape == (51, 10824),
}

validity_checks_630v18ir1 = {
    "prediction_finite":
        bool(
            np.isfinite(
                pred_630v18ir1
            ).all()
        ),

    "mc_std_finite":
        bool(
            np.isfinite(
                std_630v18ir1
            ).all()
        ),

    "mc_std_nonnegative":
        bool(
            np.all(
                std_630v18ir1 >= 0.0
            )
        ),

    "risk_finite":
        bool(
            np.isfinite(
                risk_630v18ir1
            ).all()
        ),

    "risk_in_unit_interval":
        bool(
            np.all(
                (risk_630v18ir1 >= 0.0)
                &
                (risk_630v18ir1 <= 1.0)
            )
        ),
}

if not all(
    shape_checks_630v18ir1.values()
):
    raise RuntimeError(
        "Noise retained-array shape changed."
    )

if not all(
    validity_checks_630v18ir1.values()
):
    raise RuntimeError(
        "Noise retained-array validity check failed."
    )


# =============================================================================
# 2) RECOMPUTE SCIENTIFIC HASHES AGAINST RESIDENT AUTHORITIES
#
# NO manually copied hashes are used.
# =============================================================================

resident_authority_hashes_630v18ir1 = {
    "prediction":
        str(
            FD004_NOISE_BLOCK_PREDICTIONS_SHA256_630V18H
        ),

    "mc_std":
        str(
            FD004_NOISE_BLOCK_MC_STD_SHA256_630V18H
        ),

    "final_risk":
        str(
            FD004_NOISE_BLOCK_FINAL_RISK_SHA256_630V18H
        ),

    "block_contract":
        str(
            FD004_NOISE_BLOCK_CONTRACT_SHA256_630V18H
        ),

    "checkpoint_npz":
        str(
            FD004_NOISE_CHECKPOINT_NPZ_SHA256_630V18H
        ),

    "checkpoint_json":
        str(
            FD004_NOISE_CHECKPOINT_JSON_SHA256_630V18H
        ),
}


# SHA-256 structural sanity check.
for (
    key_630v18ir1,
    value_630v18ir1,
) in resident_authority_hashes_630v18ir1.items():

    if (
        len(
            value_630v18ir1
        )
        != 64
    ):
        raise RuntimeError(
            f"Resident authority {key_630v18ir1} "
            "is not a 64-character SHA-256."
        )


rehash_checks_630v18ir1 = {
    "prediction":
        str(
            sha256_array_630v2(
                pred_630v18ir1
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "prediction"
        ],

    "mc_std":
        str(
            sha256_array_630v2(
                std_630v18ir1
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "mc_std"
        ],

    "final_risk":
        str(
            sha256_array_630v2(
                risk_630v18ir1
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "final_risk"
        ],
}

if not all(
    rehash_checks_630v18ir1.values()
):
    failed_630v18ir1 = [
        key
        for key, passed
        in rehash_checks_630v18ir1.items()
        if not passed
    ]

    raise RuntimeError(
        "Resident noise scientific rehash failed: "
        + ", ".join(
            failed_630v18ir1
        )
    )


# =============================================================================
# 3) MANIFEST / SCENARIO-EVIDENCE ORDER
# =============================================================================

manifest_checks_630v18ir1 = {
    "row_count":
        len(
            manifest_630v18ir1
        )
        == 51,

    "family_noise":
        bool(
            np.all(
                manifest_630v18ir1[
                    "family"
                ]
                .astype(str)
                .to_numpy()
                ==
                "noise"
            )
        ),

    "number_range":
        (
            int(
                manifest_630v18ir1[
                    "scenario_number"
                ].min()
            )
            == 130
            and
            int(
                manifest_630v18ir1[
                    "scenario_number"
                ].max()
            )
            == 189
        ),

    "numbers_strictly_increasing":
        bool(
            np.all(
                np.diff(
                    manifest_630v18ir1[
                        "scenario_number"
                    ].to_numpy(
                        dtype=np.int64
                    )
                )
                > 0
            )
        ),
}

if len(
    scenario_evidence_630v18ir1
) != 51:
    raise RuntimeError(
        "Noise scenario-evidence count changed."
    )

evidence_ids_630v18ir1 = np.asarray(
    [
        str(
            row[
                "scenario_id"
            ]
        )
        for row
        in scenario_evidence_630v18ir1
    ],
    dtype=object,
)

evidence_numbers_630v18ir1 = np.asarray(
    [
        int(
            row[
                "scenario_number"
            ]
        )
        for row
        in scenario_evidence_630v18ir1
    ],
    dtype=np.int64,
)

evidence_order_exact_630v18ir1 = bool(
    np.array_equal(
        evidence_ids_630v18ir1,
        manifest_630v18ir1[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
    and
    np.array_equal(
        evidence_numbers_630v18ir1,
        manifest_630v18ir1[
            "scenario_number"
        ].to_numpy(
            dtype=np.int64
        ),
    )
)

if not all(
    manifest_checks_630v18ir1.values()
):
    raise RuntimeError(
        "Noise manifest closure check failed."
    )

if not evidence_order_exact_630v18ir1:
    raise RuntimeError(
        "Noise scenario-evidence order changed."
    )


mc_hash_checks_630v18ir1 = {
    "count_51":
        len(
            mc_hashes_630v18ir1
        )
        == 51,

    "format_exact":
        bool(
            all(
                len(value) == 64
                and
                all(
                    char in "0123456789abcdef"
                    for char in value.lower()
                )
                for value in mc_hashes_630v18ir1
            )
        ),
}

if not all(
    mc_hash_checks_630v18ir1.values()
):
    raise RuntimeError(
        "Noise full-MC hash evidence changed."
    )


# =============================================================================
# 4) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18ir1 = dict(
    FD004_NOISE_BLOCK_CONTRACT_630V18H
)

contract_recomputed_sha_630v18ir1 = (
    hashlib.sha256(
        json.dumps(
            contract_630v18ir1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

contract_checks_630v18ir1 = {
    "sha_exact":
        contract_recomputed_sha_630v18ir1
        ==
        resident_authority_hashes_630v18ir1[
            "block_contract"
        ],

    "family":
        str(
            contract_630v18ir1.get(
                "family"
            )
        )
        ==
        "noise",

    "scenario_count":
        int(
            contract_630v18ir1.get(
                "scenario_count"
            )
        )
        ==
        51,

    "prediction_hash":
        str(
            contract_630v18ir1.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "prediction"
        ],

    "mc_std_hash":
        str(
            contract_630v18ir1.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "mc_std"
        ],

    "risk_hash":
        str(
            contract_630v18ir1.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "final_risk"
        ],

    "bias_closure_link":
        str(
            contract_630v18ir1.get(
                "bias_final_closure_sha256"
            )
        )
        ==
        str(
            FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G
        ),

    "full_mc_not_retained":
        bool(
            contract_630v18ir1.get(
                "full_mc_matrices_retained"
            )
        )
        is False,

    "base_unchanged":
        bool(
            contract_630v18ir1.get(
                "corruption_base_unchanged"
            )
        )
        is True,

    "model_state_unchanged":
        bool(
            contract_630v18ir1.get(
                "model_state_unchanged"
            )
        )
        is True,

    "model_mode_unchanged":
        bool(
            contract_630v18ir1.get(
                "model_mode_unchanged"
            )
        )
        is True,

    "dev_outcomes_unused":
        bool(
            contract_630v18ir1.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18ir1.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            contract_630v18ir1.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18ir1.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18ir1.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(
    contract_checks_630v18ir1.values()
):
    failed_630v18ir1 = [
        key
        for key, passed
        in contract_checks_630v18ir1.items()
        if not passed
    ]

    raise RuntimeError(
        "Noise block-contract closure failed: "
        + ", ".join(
            failed_630v18ir1
        )
    )


# =============================================================================
# 5) CHECKPOINT BYTE HASHES AGAINST RESIDENT AUTHORITIES
# =============================================================================

npz_path_630v18ir1 = Path(
    FD004_NOISE_CHECKPOINT_NPZ_PATH_630V18H
)

json_path_630v18ir1 = Path(
    FD004_NOISE_CHECKPOINT_JSON_PATH_630V18H
)

if not npz_path_630v18ir1.is_file():
    raise RuntimeError(
        "Noise NPZ checkpoint file is missing."
    )

if not json_path_630v18ir1.is_file():
    raise RuntimeError(
        "Noise JSON checkpoint file is missing."
    )


def _file_sha256_630v18ir1(path):
    h_630v18ir1 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_630v18ir1:

        while True:
            chunk_630v18ir1 = (
                file_630v18ir1.read(
                    1024 * 1024
                )
            )

            if not chunk_630v18ir1:
                break

            h_630v18ir1.update(
                chunk_630v18ir1
            )

    return h_630v18ir1.hexdigest()


checkpoint_byte_checks_630v18ir1 = {
    "npz":
        _file_sha256_630v18ir1(
            npz_path_630v18ir1
        )
        ==
        resident_authority_hashes_630v18ir1[
            "checkpoint_npz"
        ],

    "json":
        _file_sha256_630v18ir1(
            json_path_630v18ir1
        )
        ==
        resident_authority_hashes_630v18ir1[
            "checkpoint_json"
        ],
}

if not all(
    checkpoint_byte_checks_630v18ir1.values()
):
    raise RuntimeError(
        "Noise checkpoint byte hash changed."
    )


# =============================================================================
# 6) REOPEN SAFE NPZ WITH allow_pickle=False AND REVERIFY
# =============================================================================

expected_npz_keys_630v18ir1 = tuple(
    sorted(
        (
            "scenario_number",
            "scenario_id",
            "deterministic_prediction",
            "mc_std",
            "final_selective_risk",
            "full_mc_matrix_sha256",
        )
    )
)

with np.load(
    npz_path_630v18ir1,
    allow_pickle=False,
) as npz_630v18ir1:

    keys_630v18ir1 = tuple(
        sorted(
            npz_630v18ir1.files
        )
    )

    if (
        keys_630v18ir1
        !=
        expected_npz_keys_630v18ir1
    ):
        raise RuntimeError(
            "Noise NPZ key set changed."
        )

    loaded_numbers_630v18ir1 = np.asarray(
        npz_630v18ir1[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    loaded_id_raw_630v18ir1 = np.asarray(
        npz_630v18ir1[
            "scenario_id"
        ]
    )

    loaded_ids_630v18ir1 = (
        loaded_id_raw_630v18ir1
        .astype(str)
        .reshape(-1)
    )

    loaded_pred_630v18ir1 = np.asarray(
        npz_630v18ir1[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    loaded_std_630v18ir1 = np.asarray(
        npz_630v18ir1[
            "mc_std"
        ],
        dtype=np.float64,
    )

    loaded_risk_630v18ir1 = np.asarray(
        npz_630v18ir1[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    loaded_mc_hashes_630v18ir1 = tuple(
        str(x)
        for x in np.asarray(
            npz_630v18ir1[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


npz_checks_630v18ir1 = {
    "scenario_id_non_object":
        bool(
            not loaded_id_raw_630v18ir1.dtype.hasobject
        ),

    "scenario_number":
        bool(
            np.array_equal(
                loaded_numbers_630v18ir1,
                manifest_630v18ir1[
                    "scenario_number"
                ].to_numpy(
                    dtype=np.int64
                ),
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                loaded_ids_630v18ir1,
                manifest_630v18ir1[
                    "scenario_id"
                ].astype(str).to_numpy(),
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                loaded_pred_630v18ir1,
                pred_630v18ir1,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                loaded_std_630v18ir1,
                std_630v18ir1,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                loaded_risk_630v18ir1,
                risk_630v18ir1,
            )
        ),

    "full_mc_hashes":
        loaded_mc_hashes_630v18ir1
        ==
        mc_hashes_630v18ir1,
}

if not all(
    npz_checks_630v18ir1.values()
):
    failed_630v18ir1 = [
        key
        for key, passed
        in npz_checks_630v18ir1.items()
        if not passed
    ]

    raise RuntimeError(
        "Noise NPZ safe round-trip failed: "
        + ", ".join(
            failed_630v18ir1
        )
    )


# =============================================================================
# 7) JSON CHECKPOINT REVERIFY
# =============================================================================

with open(
    json_path_630v18ir1,
    "r",
    encoding="utf-8",
) as file_630v18ir1:

    json_loaded_630v18ir1 = json.load(
        file_630v18ir1
    )


json_checks_630v18ir1 = {
    "dataset":
        str(
            json_loaded_630v18ir1.get(
                "dataset"
            )
        )
        ==
        "FD004",

    "stage":
        str(
            json_loaded_630v18ir1.get(
                "stage"
            )
        )
        ==
        "6.30V-18H",

    "family":
        str(
            json_loaded_630v18ir1.get(
                "family"
            )
        )
        ==
        "noise",

    "scenario_count":
        int(
            json_loaded_630v18ir1.get(
                "scenario_count"
            )
        )
        ==
        51,

    "npz_sha":
        str(
            json_loaded_630v18ir1.get(
                "npz_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "checkpoint_npz"
        ],

    "block_contract_sha":
        str(
            json_loaded_630v18ir1.get(
                "block_contract_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "block_contract"
        ],

    "prediction_sha":
        str(
            json_loaded_630v18ir1.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "prediction"
        ],

    "mc_std_sha":
        str(
            json_loaded_630v18ir1.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "mc_std"
        ],

    "risk_sha":
        str(
            json_loaded_630v18ir1.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18ir1[
            "final_risk"
        ],

    "scenario_evidence_count":
        len(
            json_loaded_630v18ir1.get(
                "scenario_evidence",
                [],
            )
        )
        ==
        51,

    "dev_outcomes_unused":
        bool(
            json_loaded_630v18ir1.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            json_loaded_630v18ir1.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            json_loaded_630v18ir1.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            json_loaded_630v18ir1.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            json_loaded_630v18ir1.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(
    json_checks_630v18ir1.values()
):
    failed_630v18ir1 = [
        key
        for key, passed
        in json_checks_630v18ir1.items()
        if not passed
    ]

    raise RuntimeError(
        "Noise JSON verification failed: "
        + ", ".join(
            failed_630v18ir1
        )
    )


# =============================================================================
# 8) CONFIRM PRIOR 18H ROUND-TRIP FLAGS TOO
# =============================================================================

prior_roundtrip_checks_630v18ir1 = dict(
    FD004_NOISE_CHECKPOINT_ROUNDTRIP_CHECKS_630V18H
)

prior_roundtrip_all_true_630v18ir1 = bool(
    prior_roundtrip_checks_630v18ir1
    and
    all(
        bool(value)
        for value
        in prior_roundtrip_checks_630v18ir1.values()
    )
)

if not prior_roundtrip_all_true_630v18ir1:
    raise RuntimeError(
        "Prior 18H checkpoint round-trip evidence changed."
    )


# =============================================================================
# 9) FINAL NOISE CLOSURE
# =============================================================================

FD004_NOISE_FINAL_CLOSURE_630V18IR1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18I-R1",

    "family":
        "noise",

    "status":
        "NOISE_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        51,

    "prediction_block_sha256":
        resident_authority_hashes_630v18ir1[
            "prediction"
        ],

    "mc_std_block_sha256":
        resident_authority_hashes_630v18ir1[
            "mc_std"
        ],

    "final_risk_block_sha256":
        resident_authority_hashes_630v18ir1[
            "final_risk"
        ],

    "noise_block_contract_sha256":
        resident_authority_hashes_630v18ir1[
            "block_contract"
        ],

    "safe_checkpoint_npz_sha256":
        resident_authority_hashes_630v18ir1[
            "checkpoint_npz"
        ],

    "safe_checkpoint_json_sha256":
        resident_authority_hashes_630v18ir1[
            "checkpoint_json"
        ],

    "safe_checkpoint_roundtrip_exact":
        True,

    "full_mc_matrix_hash_count":
        len(
            mc_hashes_630v18ir1
        ),

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1 = (
    hashlib.sha256(
        json.dumps(
            FD004_NOISE_FINAL_CLOSURE_630V18IR1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_NOISE_FAMILY_CLOSED_630V18IR1 = True

FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18IR1 = False
FD004_DEV_TARGETS_ERRORS_USED_630V18IR1 = False
FD004_THRESHOLD_SELECTED_630V18IR1 = False
FD004_POLICY_SELECTED_630V18IR1 = False
FD004_TEST_CONTENTS_READ_630V18IR1 = False
FD004_RUL_CONTENTS_READ_630V18IR1 = False

all_checks_passed_630v18ir1 = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18I-R1 — FD004 NOISE FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Noise scenarios rerun             :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- RESIDENT SCIENTIFIC EVIDENCE ---")

print(
    "Prediction rehash exact           :",
    rehash_checks_630v18ir1[
        "prediction"
    ],
)

print(
    "MC STD rehash exact               :",
    rehash_checks_630v18ir1[
        "mc_std"
    ],
)

print(
    "Final-risk rehash exact           :",
    rehash_checks_630v18ir1[
        "final_risk"
    ],
)

print(
    "Manifest checks all true          :",
    all(
        manifest_checks_630v18ir1.values()
    ),
)

print(
    "Scenario evidence order exact     :",
    evidence_order_exact_630v18ir1,
)

print(
    "Full-MC hash evidence exact       :",
    all(
        mc_hash_checks_630v18ir1.values()
    ),
)


print("\n--- BLOCK CONTRACT ---")

for key, value in (
    contract_checks_630v18ir1.items()
):
    print(
        f"{key:28s}:",
        value,
    )


print("\n--- SAFE CHECKPOINT ---")

print(
    "NPZ byte SHA exact                :",
    checkpoint_byte_checks_630v18ir1[
        "npz"
    ],
)

print(
    "JSON byte SHA exact               :",
    checkpoint_byte_checks_630v18ir1[
        "json"
    ],
)

print(
    "NPZ allow_pickle=False checks     :",
    npz_checks_630v18ir1,
)

print(
    "JSON checks                       :",
    json_checks_630v18ir1,
)

print(
    "Prior 18H round-trip all true     :",
    prior_roundtrip_all_true_630v18ir1,
)


print("\n--- FINAL CLOSURE ---")

print(
    "Noise family closed               :",
    FD004_NOISE_FAMILY_CLOSED_630V18IR1,
)

print(
    "Noise final closure SHA-256       :",
    FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1,
)


print(
    "\nPASS — FD004 noise is evidence-locked using only resident "
    "authoritative hashes and direct recomputation. No manually "
    "transcribed result hashes were used."
)

print(
    "Next: execute only the 102 admissible drift scenarios."
)

In [ ]:
# STEP 6.30V-18J — FD004 DRIFT FAMILY EXECUTION
# 102 admissible drift scenarios only.
#
# LABEL-FREE:
# corruption -> normalization -> windows -> deterministic prediction
# -> 50-pass MC dropout -> reliability chain
#
# NO DEV targets/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.
#
# Retain:
# - deterministic prediction
# - MC STD
# - final selective risk
# - SHA-256 of each full MC matrix, then discard full MC matrix.
#
# Checkpoint:
# - fixed-width Unicode scenario IDs
# - allow_pickle=False round-trip verification
# - persistence failure does NOT invalidate completed scientific execution.

import numpy as np
import pandas as pd
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18j = (
    "all_checks_passed_630v18ir1",
    "FD004_NOISE_FAMILY_CLOSED_630V18IR1",
    "FD004_NOISE_FINAL_CLOSURE_630V18IR1",
    "FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_DETERMINISTIC_PREDICTOR_630V18CR1",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
    "FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0",
)

missing_630v18j = [
    name
    for name in required_630v18j
    if name not in globals()
]

if missing_630v18j:
    raise RuntimeError(
        "Missing STEP 6.30V-18J authority object(s): "
        + ", ".join(missing_630v18j)
    )

if not bool(all_checks_passed_630v18ir1):
    raise RuntimeError(
        "STEP 6.30V-18I-R1 must remain passed."
    )

if not bool(FD004_NOISE_FAMILY_CLOSED_630V18IR1):
    raise RuntimeError(
        "Noise family must remain evidence-locked before drift."
    )


# Verify resident noise closure without any manually copied hash.
noise_closure_recomputed_sha_630v18j = hashlib.sha256(
    json.dumps(
        FD004_NOISE_FINAL_CLOSURE_630V18IR1,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    noise_closure_recomputed_sha_630v18j
    !=
    str(
        FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1
    )
):
    raise RuntimeError(
        "Resident noise final-closure evidence changed."
    )


# =============================================================================
# 1) FREEZE DRIFT BLOCK FROM EXECUTION MANIFEST
# =============================================================================

execution_manifest_630v18j = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

drift_manifest_630v18j = (
    execution_manifest_630v18j.loc[
        execution_manifest_630v18j[
            "family"
        ].astype(str)
        == "drift"
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(drift_manifest_630v18j) != 102:
    raise RuntimeError(
        "Expected exactly 102 admissible drift scenarios."
    )

drift_numbers_630v18j = (
    drift_manifest_630v18j[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

if (
    int(drift_numbers_630v18j.min()) != 196
    or
    int(drift_numbers_630v18j.max()) != 315
):
    raise RuntimeError(
        "Frozen drift scenario-number range changed."
    )

if not np.all(
    drift_manifest_630v18j[
        "family"
    ].astype(str).to_numpy()
    ==
    "drift"
):
    raise RuntimeError(
        "Non-drift row entered the drift block."
    )

if not (
    drift_manifest_630v18j[
        "alpha"
    ].notna().all()
    and
    drift_manifest_630v18j[
        "direction"
    ].notna().all()
):
    raise RuntimeError(
        "Drift manifest has missing alpha/direction values."
    )

if not (
    drift_manifest_630v18j[
        "direction"
    ]
    .astype(float)
    .isin(
        [-1.0, 1.0]
    )
    .all()
):
    raise RuntimeError(
        "Drift manifest contains an invalid direction."
    )


# =============================================================================
# 2) FROZEN EXECUTION AUTHORITIES
# =============================================================================

base_630v18j = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v18j = (
    base_630v18j.copy(deep=True)
)

regime_ids_630v18j = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

expected_window_units_630v18j = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

expected_endpoint_cycles_630v18j = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if expected_window_units_630v18j.shape != (
    10824,
):
    raise RuntimeError(
        "Frozen DEV window-unit authority shape changed."
    )

if expected_endpoint_cycles_630v18j.shape != (
    10824,
):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle authority shape changed."
    )

model_630v18j = (
    MODEL_FD004_630I
)

device_630v18j = (
    FD004_DEVICE_630I
)

if bool(
    model_630v18j.training
):
    raise RuntimeError(
        "FD004 model must be in eval mode before drift execution."
    )


def _model_state_sha_630v18j(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(
        state.keys()
    ):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        arr = tensor.numpy()

        h.update(
            key.encode("utf-8")
        )

        h.update(
            str(
                arr.dtype
            ).encode("utf-8")
        )

        h.update(
            str(
                tuple(
                    arr.shape
                )
            ).encode("utf-8")
        )

        h.update(
            arr.tobytes(
                order="C"
            )
        )

    return h.hexdigest()


model_state_before_630v18j = (
    _model_state_sha_630v18j(
        model_630v18j
    )
)

model_mode_before_630v18j = bool(
    model_630v18j.training
)


# =============================================================================
# 3) PREALLOCATE ONLY FROZEN RETAINED PRODUCTS
# =============================================================================

n_scenarios_630v18j = 102
n_windows_630v18j = 10824

drift_pred_matrix_630v18j = np.empty(
    (
        n_scenarios_630v18j,
        n_windows_630v18j,
    ),
    dtype=np.float32,
)

drift_mc_std_matrix_630v18j = np.empty(
    (
        n_scenarios_630v18j,
        n_windows_630v18j,
    ),
    dtype=np.float64,
)

drift_final_risk_matrix_630v18j = np.empty(
    (
        n_scenarios_630v18j,
        n_windows_630v18j,
    ),
    dtype=np.float64,
)

drift_mc_matrix_hashes_630v18j = []
drift_scenario_evidence_630v18j = []


# =============================================================================
# 4) EXECUTE EXACTLY 102 ADMISSIBLE DRIFT SCENARIOS
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18J — FD004 DRIFT FAMILY EXECUTION "
    "(102 ADMISSIBLE SCENARIOS)"
)

print("=" * 176)

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nStarting frozen drift block        :",
    n_scenarios_630v18j,
    "scenarios",
)


for row_index_630v18j, row_630v18j in (
    drift_manifest_630v18j.iterrows()
):

    scenario_id_630v18j = str(
        row_630v18j[
            "scenario_id"
        ]
    )

    scenario_number_630v18j = int(
        row_630v18j[
            "scenario_number"
        ]
    )

    sensor_630v18j = str(
        row_630v18j[
            "sensor"
        ]
    )

    alpha_630v18j = float(
        row_630v18j[
            "alpha"
        ]
    )

    direction_630v18j = int(
        row_630v18j[
            "direction"
        ]
    )

    # -------------------------------------------------------------------------
    # A) Frozen corruption dispatcher
    # -------------------------------------------------------------------------

    corrupted_raw_630v18j = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v18j,
            row_630v18j,
        )
    )

    if not isinstance(
        corrupted_raw_630v18j,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: dispatcher did not return DataFrame."
        )

    if corrupted_raw_630v18j.shape != (
        12274,
        27,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: corrupted raw shape changed."
        )

    if not np.array_equal(
        corrupted_raw_630v18j[
            "regime_id"
        ].to_numpy(
            dtype=np.int64
        ),
        regime_ids_630v18j,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: regime_id changed under drift."
        )

    # -------------------------------------------------------------------------
    # B) Frozen normalization
    # -------------------------------------------------------------------------

    normalized_630v18j = (
        normalize_fd004_raw_630v2(
            corrupted_raw_630v18j,
            regime_ids_630v18j,
        )
    )

    # -------------------------------------------------------------------------
    # C) Frozen windows
    # -------------------------------------------------------------------------

    (
        X_630v18j,
        unit_ids_630v18j,
        endpoint_cycles_630v18j,
    ) = build_windows_fd004_630v2(
        normalized_630v18j
    )

    X_630v18j = np.asarray(
        X_630v18j,
        dtype=np.float32,
    )

    if X_630v18j.shape != (
        10824,
        30,
        24,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: window tensor shape changed."
        )

    if not np.isfinite(
        X_630v18j
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18j}: non-finite model input."
        )

    if not np.array_equal(
        np.asarray(
            unit_ids_630v18j,
            dtype=np.int64,
        ).reshape(-1),
        expected_window_units_630v18j,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: window-unit membership changed."
        )

    if not np.array_equal(
        np.asarray(
            endpoint_cycles_630v18j,
            dtype=np.int64,
        ).reshape(-1),
        expected_endpoint_cycles_630v18j,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: endpoint-cycle alignment changed."
        )

    # -------------------------------------------------------------------------
    # D) Frozen deterministic inference
    # -------------------------------------------------------------------------

    deterministic_pred_630v18j = np.asarray(
        FD004_DETERMINISTIC_PREDICTOR_630V18CR1(
            X_630v18j,
            model_630v18j,
            device_630v18j,
            batch_size=512,
        ),
        dtype=np.float32,
    ).reshape(-1)

    if deterministic_pred_630v18j.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: deterministic prediction shape changed."
        )

    if not np.isfinite(
        deterministic_pred_630v18j
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18j}: non-finite deterministic prediction."
        )

    # -------------------------------------------------------------------------
    # E) Frozen 50-pass MC
    # -------------------------------------------------------------------------

    (
        mc_matrix_630v18j,
        mc_mean_630v18j,
        mc_std_630v18j,
    ) = fd004_mc_dropout_predict_630v4r2(
        X_630v18j,
        model_630v18j,
        device_630v18j,
    )

    mc_matrix_630v18j = np.asarray(
        mc_matrix_630v18j,
        dtype=np.float32,
    )

    mc_std_630v18j = np.asarray(
        mc_std_630v18j,
        dtype=np.float64,
    ).reshape(-1)

    if mc_matrix_630v18j.shape != (
        50,
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: MC matrix shape changed."
        )

    if mc_std_630v18j.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: MC STD shape changed."
        )

    if not (
        np.isfinite(
            mc_matrix_630v18j
        ).all()
        and
        np.isfinite(
            mc_std_630v18j
        ).all()
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: non-finite MC output."
        )

    if np.any(
        mc_std_630v18j < 0.0
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: negative MC STD."
        )

    mc_matrix_sha_630v18j = str(
        sha256_array_630v2(
            mc_matrix_630v18j
        )
    )

    # -------------------------------------------------------------------------
    # F) Frozen reliability chain
    # -------------------------------------------------------------------------

    (
        phase_ids_630v18j,
        adjusted_630v18j,
        g1_630v18j,
        final_risk_630v18j,
    ) = apply_fd004_final_reliability_chain_630v4r2(
        deterministic_pred_630v18j,
        mc_std_630v18j,
    )

    final_risk_630v18j = np.asarray(
        final_risk_630v18j,
        dtype=np.float64,
    ).reshape(-1)

    if final_risk_630v18j.shape != (
        10824,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: final risk shape changed."
        )

    if not np.isfinite(
        final_risk_630v18j
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18j}: non-finite final risk."
        )

    if np.any(
        (final_risk_630v18j < 0.0)
        |
        (final_risk_630v18j > 1.0)
    ):
        raise RuntimeError(
            f"{scenario_id_630v18j}: final risk escaped [0,1]."
        )

    # -------------------------------------------------------------------------
    # G) Retain frozen required products
    # -------------------------------------------------------------------------

    drift_pred_matrix_630v18j[
        row_index_630v18j
    ] = deterministic_pred_630v18j

    drift_mc_std_matrix_630v18j[
        row_index_630v18j
    ] = mc_std_630v18j

    drift_final_risk_matrix_630v18j[
        row_index_630v18j
    ] = final_risk_630v18j

    drift_mc_matrix_hashes_630v18j.append(
        mc_matrix_sha_630v18j
    )

    drift_scenario_evidence_630v18j.append(
        {
            "scenario_id":
                scenario_id_630v18j,

            "scenario_number":
                scenario_number_630v18j,

            "family":
                "drift",

            "sensor":
                sensor_630v18j,

            "alpha":
                alpha_630v18j,

            "direction":
                direction_630v18j,

            "deterministic_prediction_sha256":
                str(
                    sha256_array_630v2(
                        deterministic_pred_630v18j
                    )
                ),

            "mc_matrix_sha256":
                mc_matrix_sha_630v18j,

            "mc_std_sha256":
                str(
                    sha256_array_630v2(
                        mc_std_630v18j
                    )
                ),

            "final_selective_risk_sha256":
                str(
                    sha256_array_630v2(
                        final_risk_630v18j
                    )
                ),
        }
    )

    # Discard products not retained by the frozen contract.
    del mc_matrix_630v18j
    del mc_mean_630v18j

    del phase_ids_630v18j
    del adjusted_630v18j
    del g1_630v18j

    del corrupted_raw_630v18j
    del normalized_630v18j
    del X_630v18j

    if (
        (row_index_630v18j + 1) == 1
        or
        (row_index_630v18j + 1) % 10 == 0
        or
        (row_index_630v18j + 1)
        ==
        n_scenarios_630v18j
    ):
        print(
            f"Completed drift scenario "
            f"{row_index_630v18j + 1:3d}/"
            f"{n_scenarios_630v18j}: "
            f"{scenario_id_630v18j}"
        )


# =============================================================================
# 5) POST-BLOCK SCIENTIFIC SAFETY
# =============================================================================

if not np.isfinite(
    drift_pred_matrix_630v18j
).all():
    raise RuntimeError(
        "Retained drift deterministic predictions contain non-finite values."
    )

if not np.isfinite(
    drift_mc_std_matrix_630v18j
).all():
    raise RuntimeError(
        "Retained drift MC STD contains non-finite values."
    )

if not np.isfinite(
    drift_final_risk_matrix_630v18j
).all():
    raise RuntimeError(
        "Retained drift final risk contains non-finite values."
    )

if np.any(
    drift_mc_std_matrix_630v18j < 0.0
):
    raise RuntimeError(
        "Retained drift MC STD contains negative values."
    )

if np.any(
    (drift_final_risk_matrix_630v18j < 0.0)
    |
    (drift_final_risk_matrix_630v18j > 1.0)
):
    raise RuntimeError(
        "Retained drift risk escaped [0,1]."
    )

if len(
    drift_mc_matrix_hashes_630v18j
) != 102:
    raise RuntimeError(
        "Drift MC matrix hash count changed."
    )

if len(
    drift_scenario_evidence_630v18j
) != 102:
    raise RuntimeError(
        "Drift scenario evidence count changed."
    )

base_unchanged_630v18j = bool(
    base_630v18j.equals(
        base_snapshot_630v18j
    )
)

if not base_unchanged_630v18j:
    raise RuntimeError(
        "Frozen corruption base changed during drift execution."
    )

model_state_after_630v18j = (
    _model_state_sha_630v18j(
        model_630v18j
    )
)

model_mode_after_630v18j = bool(
    model_630v18j.training
)

model_state_unchanged_630v18j = bool(
    model_state_before_630v18j
    ==
    model_state_after_630v18j
)

model_mode_unchanged_630v18j = bool(
    model_mode_before_630v18j
    ==
    model_mode_after_630v18j
)

if not model_state_unchanged_630v18j:
    raise RuntimeError(
        "Model state changed during drift block."
    )

if not model_mode_unchanged_630v18j:
    raise RuntimeError(
        "Model mode changed during drift block."
    )


# =============================================================================
# 6) FREEZE SCIENTIFIC DRIFT EVIDENCE
# =============================================================================

FD004_DRIFT_SCENARIO_MANIFEST_630V18J = (
    drift_manifest_630v18j.copy()
)

FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J = (
    drift_pred_matrix_630v18j
)

FD004_DRIFT_MC_STD_630V18J = (
    drift_mc_std_matrix_630v18j
)

FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J = (
    drift_final_risk_matrix_630v18j
)

FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J = tuple(
    drift_mc_matrix_hashes_630v18j
)

FD004_DRIFT_SCENARIO_EVIDENCE_630V18J = tuple(
    drift_scenario_evidence_630v18j
)


FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J = str(
    sha256_array_630v2(
        FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J
    )
)

FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J = str(
    sha256_array_630v2(
        FD004_DRIFT_MC_STD_630V18J
    )
)

FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J = str(
    sha256_array_630v2(
        FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J
    )
)


drift_block_contract_630v18j = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18J",

    "family":
        "drift",

    "status":
        "DRIFT_BLOCK_COMPLETE_LABEL_FREE",

    "scenario_count":
        102,

    "scenario_number_min":
        int(
            drift_numbers_630v18j.min()
        ),

    "scenario_number_max":
        int(
            drift_numbers_630v18j.max()
        ),

    "deterministic_prediction_shape":
        tuple(
            FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J.shape
        ),

    "mc_std_shape":
        tuple(
            FD004_DRIFT_MC_STD_630V18J.shape
        ),

    "final_selective_risk_shape":
        tuple(
            FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J.shape
        ),

    "full_mc_matrix_hash_count":
        len(
            FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J
        ),

    "full_mc_matrices_retained":
        False,

    "prediction_block_sha256":
        FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J,

    "mc_std_block_sha256":
        FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J,

    "final_risk_block_sha256":
        FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J,

    "noise_final_closure_sha256":
        str(
            FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1
        ),

    "admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helper_return_audit_sha256":
        str(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0
        ),

    "corruption_base_unchanged":
        base_unchanged_630v18j,

    "model_state_unchanged":
        model_state_unchanged_630v18j,

    "model_mode_unchanged":
        model_mode_unchanged_630v18j,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_DRIFT_BLOCK_CONTRACT_630V18J = (
    drift_block_contract_630v18j
)

FD004_DRIFT_BLOCK_CONTRACT_SHA256_630V18J = (
    hashlib.sha256(
        json.dumps(
            drift_block_contract_630v18j,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) SAFE CHECKPOINT — FIXED-WIDTH UNICODE FROM THE START
# =============================================================================

checkpoint_saved_630v18j = False
checkpoint_roundtrip_exact_630v18j = False
checkpoint_error_630v18j = None

checkpoint_npz_path_630v18j = None
checkpoint_json_path_630v18j = None

checkpoint_npz_sha_630v18j = None
checkpoint_json_sha_630v18j = None

checkpoint_roundtrip_checks_630v18j = None

try:
    checkpoint_dir_630v18j = Path(
        r"C:\RUL_Reliability_Research\checkpoints"
    )

    checkpoint_dir_630v18j.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_npz_path_630v18j = (
        checkpoint_dir_630v18j
        /
        "fd004_drift_block_630v18j_v1_safe.npz"
    )

    checkpoint_json_path_630v18j = (
        checkpoint_dir_630v18j
        /
        "fd004_drift_block_630v18j_v1_safe.json"
    )

    npz_tmp_630v18j = (
        checkpoint_dir_630v18j
        /
        "fd004_drift_block_630v18j_v1_safe.npz.tmp"
    )

    json_tmp_630v18j = (
        checkpoint_dir_630v18j
        /
        "fd004_drift_block_630v18j_v1_safe.json.tmp"
    )

    scenario_id_list_630v18j = (
        drift_manifest_630v18j[
            "scenario_id"
        ]
        .astype(str)
        .tolist()
    )

    scenario_id_max_len_630v18j = max(
        len(value)
        for value
        in scenario_id_list_630v18j
    )

    scenario_id_unicode_630v18j = np.asarray(
        scenario_id_list_630v18j,
        dtype=f"<U{scenario_id_max_len_630v18j}",
    )

    full_mc_hash_unicode_630v18j = np.asarray(
        FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J,
        dtype="<U64",
    )

    with open(
        npz_tmp_630v18j,
        "wb",
    ) as file:

        np.savez_compressed(
            file,

            scenario_number=
                drift_manifest_630v18j[
                    "scenario_number"
                ].to_numpy(
                    dtype=np.int64
                ),

            scenario_id=
                scenario_id_unicode_630v18j,

            deterministic_prediction=
                FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J,

            mc_std=
                FD004_DRIFT_MC_STD_630V18J,

            final_selective_risk=
                FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J,

            full_mc_matrix_sha256=
                full_mc_hash_unicode_630v18j,
        )

    os.replace(
        npz_tmp_630v18j,
        checkpoint_npz_path_630v18j,
    )


    def _file_sha256_630v18j(path):
        h = hashlib.sha256()

        with open(
            path,
            "rb",
        ) as file:

            while True:
                chunk = file.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                h.update(
                    chunk
                )

        return h.hexdigest()


    checkpoint_npz_sha_630v18j = (
        _file_sha256_630v18j(
            checkpoint_npz_path_630v18j
        )
    )


    checkpoint_metadata_630v18j = {
        "dataset":
            "FD004",

        "stage":
            "6.30V-18J",

        "family":
            "drift",

        "status":
            "SAFE_SERIALIZATION_CHECKPOINT",

        "scenario_count":
            102,

        "scenario_id_dtype":
            str(
                scenario_id_unicode_630v18j.dtype
            ),

        "block_contract_sha256":
            FD004_DRIFT_BLOCK_CONTRACT_SHA256_630V18J,

        "prediction_block_sha256":
            FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J,

        "mc_std_block_sha256":
            FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J,

        "final_risk_block_sha256":
            FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J,

        "npz_sha256":
            checkpoint_npz_sha_630v18j,

        "scenario_evidence":
            list(
                FD004_DRIFT_SCENARIO_EVIDENCE_630V18J
            ),

        "dev_targets_errors_used":
            False,

        "threshold_selected":
            False,

        "policy_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    }


    with open(
        json_tmp_630v18j,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            checkpoint_metadata_630v18j,
            file,
            sort_keys=True,
            indent=2,
            default=str,
        )

    os.replace(
        json_tmp_630v18j,
        checkpoint_json_path_630v18j,
    )

    checkpoint_json_sha_630v18j = (
        _file_sha256_630v18j(
            checkpoint_json_path_630v18j
        )
    )


    # -------------------------------------------------------------------------
    # Immediate safe round-trip with allow_pickle=False
    # -------------------------------------------------------------------------

    with np.load(
        checkpoint_npz_path_630v18j,
        allow_pickle=False,
    ) as checkpoint_npz_630v18j:

        loaded_numbers_630v18j = np.asarray(
            checkpoint_npz_630v18j[
                "scenario_number"
            ],
            dtype=np.int64,
        ).reshape(-1)

        loaded_id_raw_630v18j = np.asarray(
            checkpoint_npz_630v18j[
                "scenario_id"
            ]
        )

        loaded_ids_630v18j = (
            loaded_id_raw_630v18j
            .astype(str)
            .reshape(-1)
        )

        loaded_pred_630v18j = np.asarray(
            checkpoint_npz_630v18j[
                "deterministic_prediction"
            ],
            dtype=np.float32,
        )

        loaded_std_630v18j = np.asarray(
            checkpoint_npz_630v18j[
                "mc_std"
            ],
            dtype=np.float64,
        )

        loaded_risk_630v18j = np.asarray(
            checkpoint_npz_630v18j[
                "final_selective_risk"
            ],
            dtype=np.float64,
        )

        loaded_mc_hashes_630v18j = tuple(
            str(x)
            for x in np.asarray(
                checkpoint_npz_630v18j[
                    "full_mc_matrix_sha256"
                ]
            ).reshape(-1)
        )


    checkpoint_roundtrip_checks_630v18j = {
        "scenario_id_non_object":
            bool(
                not loaded_id_raw_630v18j.dtype.hasobject
            ),

        "scenario_number":
            bool(
                np.array_equal(
                    loaded_numbers_630v18j,
                    drift_manifest_630v18j[
                        "scenario_number"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                )
            ),

        "scenario_id":
            bool(
                np.array_equal(
                    loaded_ids_630v18j,
                    scenario_id_unicode_630v18j,
                )
            ),

        "prediction":
            bool(
                np.array_equal(
                    loaded_pred_630v18j,
                    FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J,
                )
            ),

        "mc_std":
            bool(
                np.array_equal(
                    loaded_std_630v18j,
                    FD004_DRIFT_MC_STD_630V18J,
                )
            ),

        "final_risk":
            bool(
                np.array_equal(
                    loaded_risk_630v18j,
                    FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J,
                )
            ),

        "full_mc_hashes":
            loaded_mc_hashes_630v18j
            ==
            tuple(
                FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J
            ),
    }

    checkpoint_roundtrip_exact_630v18j = bool(
        all(
            checkpoint_roundtrip_checks_630v18j.values()
        )
    )

    if not checkpoint_roundtrip_exact_630v18j:
        raise RuntimeError(
            "Drift safe checkpoint round-trip verification failed."
        )

    checkpoint_saved_630v18j = True

except Exception as exc_630v18j:
    checkpoint_error_630v18j = repr(
        exc_630v18j
    )


# =============================================================================
# 8) FREEZE CHECKPOINT STATUS
# =============================================================================

FD004_DRIFT_CHECKPOINT_SAVED_630V18J = bool(
    checkpoint_saved_630v18j
)

FD004_DRIFT_CHECKPOINT_ROUNDTRIP_EXACT_630V18J = bool(
    checkpoint_roundtrip_exact_630v18j
)

FD004_DRIFT_CHECKPOINT_NPZ_PATH_630V18J = (
    None
    if checkpoint_npz_path_630v18j is None
    else str(
        checkpoint_npz_path_630v18j
    )
)

FD004_DRIFT_CHECKPOINT_JSON_PATH_630V18J = (
    None
    if checkpoint_json_path_630v18j is None
    else str(
        checkpoint_json_path_630v18j
    )
)

FD004_DRIFT_CHECKPOINT_NPZ_SHA256_630V18J = (
    checkpoint_npz_sha_630v18j
)

FD004_DRIFT_CHECKPOINT_JSON_SHA256_630V18J = (
    checkpoint_json_sha_630v18j
)

FD004_DRIFT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18J = (
    checkpoint_roundtrip_checks_630v18j
)

FD004_DRIFT_CHECKPOINT_ERROR_630V18J = (
    checkpoint_error_630v18j
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD004_DRIFT_BLOCK_COMPLETE_630V18J = True
FD004_DRIFT_SCENARIOS_EXECUTED_630V18J = 102

FD004_DEV_TARGETS_ERRORS_USED_630V18J = False
FD004_THRESHOLD_SELECTED_630V18J = False
FD004_POLICY_SELECTED_630V18J = False
FD004_TEST_CONTENTS_READ_630V18J = False
FD004_RUL_CONTENTS_READ_630V18J = False

all_checks_passed_630v18j = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("\n--- DRIFT BLOCK COMPLETE ---")

print(
    "Drift scenarios executed          :",
    FD004_DRIFT_SCENARIOS_EXECUTED_630V18J,
)

print(
    "Deterministic retained shape      :",
    FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J.shape,
)

print(
    "MC STD retained shape             :",
    FD004_DRIFT_MC_STD_630V18J.shape,
)

print(
    "Final risk retained shape         :",
    FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J.shape,
)

print(
    "Full MC matrix hashes retained    :",
    len(
        FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J
    ),
)

print(
    "Full MC matrices retained         :",
    False,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18j,
)

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18j,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18j,
)

print(
    "Prediction block SHA-256          :",
    FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J,
)

print(
    "MC STD block SHA-256              :",
    FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J,
)

print(
    "Final risk block SHA-256          :",
    FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J,
)

print(
    "Drift block contract SHA-256      :",
    FD004_DRIFT_BLOCK_CONTRACT_SHA256_630V18J,
)


print("\n--- LABEL-FREE BLOCK SUMMARY ---")

print(
    "MC STD min / median / mean / max  :",
    (
        float(
            np.min(
                FD004_DRIFT_MC_STD_630V18J
            )
        ),
        float(
            np.median(
                FD004_DRIFT_MC_STD_630V18J
            )
        ),
        float(
            np.mean(
                FD004_DRIFT_MC_STD_630V18J
            )
        ),
        float(
            np.max(
                FD004_DRIFT_MC_STD_630V18J
            )
        ),
    ),
)

print(
    "Risk min / median / mean / max    :",
    (
        float(
            np.min(
                FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J
            )
        ),
        float(
            np.median(
                FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J
            )
        ),
        float(
            np.mean(
                FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J
            )
        ),
        float(
            np.max(
                FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J
            )
        ),
    ),
)


print("\n--- SAFE CHECKPOINT ---")

print(
    "Checkpoint saved                  :",
    FD004_DRIFT_CHECKPOINT_SAVED_630V18J,
)

print(
    "allow_pickle=False round-trip     :",
    FD004_DRIFT_CHECKPOINT_ROUNDTRIP_EXACT_630V18J,
)

print(
    "NPZ path                          :",
    FD004_DRIFT_CHECKPOINT_NPZ_PATH_630V18J,
)

print(
    "JSON path                         :",
    FD004_DRIFT_CHECKPOINT_JSON_PATH_630V18J,
)

print(
    "NPZ SHA-256                       :",
    FD004_DRIFT_CHECKPOINT_NPZ_SHA256_630V18J,
)

print(
    "JSON SHA-256                      :",
    FD004_DRIFT_CHECKPOINT_JSON_SHA256_630V18J,
)

print(
    "Round-trip checks                 :",
    FD004_DRIFT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18J,
)

print(
    "Checkpoint error                  :",
    FD004_DRIFT_CHECKPOINT_ERROR_630V18J,
)


print("\n--- LEAKAGE / POLICY GUARDS ---")

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print(
    "\nPASS — the 102 admissible FD004 drift scenarios are complete "
    "under the frozen label-free execution contract."
)

if (
    checkpoint_saved_630v18j
    and
    checkpoint_roundtrip_exact_630v18j
):
    print(
        "The drift block also has a verified pickle-free checkpoint. "
        "Next: close drift evidence, then proceed to dropout."
    )
else:
    print(
        "The scientific drift block is complete, but checkpoint "
        "persistence/verification failed. Do NOT rerun drift; "
        "repair persistence only."
    )

In [ ]:
# STEP 6.30V-18K — FD004 DRIFT FAMILY FINAL EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.
#
# IMPORTANT:
# No manually transcribed drift-result hashes are used.

import numpy as np
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18k = (
    "all_checks_passed_630v18j",
    "FD004_DRIFT_BLOCK_COMPLETE_630V18J",
    "FD004_DRIFT_SCENARIOS_EXECUTED_630V18J",
    "FD004_DRIFT_SCENARIO_MANIFEST_630V18J",
    "FD004_DRIFT_SCENARIO_EVIDENCE_630V18J",
    "FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J",
    "FD004_DRIFT_MC_STD_630V18J",
    "FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J",
    "FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J",
    "FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J",
    "FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J",
    "FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J",
    "FD004_DRIFT_BLOCK_CONTRACT_630V18J",
    "FD004_DRIFT_BLOCK_CONTRACT_SHA256_630V18J",
    "FD004_DRIFT_CHECKPOINT_SAVED_630V18J",
    "FD004_DRIFT_CHECKPOINT_ROUNDTRIP_EXACT_630V18J",
    "FD004_DRIFT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18J",
    "FD004_DRIFT_CHECKPOINT_NPZ_PATH_630V18J",
    "FD004_DRIFT_CHECKPOINT_JSON_PATH_630V18J",
    "FD004_DRIFT_CHECKPOINT_NPZ_SHA256_630V18J",
    "FD004_DRIFT_CHECKPOINT_JSON_SHA256_630V18J",
    "FD004_NOISE_FINAL_CLOSURE_630V18IR1",
    "FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1",
    "sha256_array_630v2",
)

missing_630v18k = [
    name
    for name in required_630v18k
    if name not in globals()
]

if missing_630v18k:
    raise RuntimeError(
        "Missing STEP 6.30V-18K authority object(s): "
        + ", ".join(missing_630v18k)
    )

if not bool(all_checks_passed_630v18j):
    raise RuntimeError(
        "STEP 6.30V-18J must remain passed."
    )

if not bool(FD004_DRIFT_BLOCK_COMPLETE_630V18J):
    raise RuntimeError(
        "Drift block is not marked complete."
    )

if int(FD004_DRIFT_SCENARIOS_EXECUTED_630V18J) != 102:
    raise RuntimeError(
        "Drift executed-scenario count changed."
    )

if not bool(FD004_DRIFT_CHECKPOINT_SAVED_630V18J):
    raise RuntimeError(
        "Drift checkpoint is not marked saved."
    )

if not bool(FD004_DRIFT_CHECKPOINT_ROUNDTRIP_EXACT_630V18J):
    raise RuntimeError(
        "Drift checkpoint is not marked round-trip exact."
    )


# =============================================================================
# 1) VERIFY PRIOR NOISE CLOSURE LINK FROM RESIDENT OBJECTS
# =============================================================================

noise_closure_recomputed_sha_630v18k = hashlib.sha256(
    json.dumps(
        FD004_NOISE_FINAL_CLOSURE_630V18IR1,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

noise_closure_link_exact_630v18k = bool(
    noise_closure_recomputed_sha_630v18k
    ==
    str(
        FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1
    )
)

if not noise_closure_link_exact_630v18k:
    raise RuntimeError(
        "Resident noise final-closure evidence changed."
    )


# =============================================================================
# 2) RESIDENT SCIENTIFIC EVIDENCE
# =============================================================================

manifest_630v18k = (
    FD004_DRIFT_SCENARIO_MANIFEST_630V18J
    .copy()
    .reset_index(drop=True)
)

pred_630v18k = np.asarray(
    FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J,
    dtype=np.float32,
)

std_630v18k = np.asarray(
    FD004_DRIFT_MC_STD_630V18J,
    dtype=np.float64,
)

risk_630v18k = np.asarray(
    FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J,
    dtype=np.float64,
)

mc_hashes_630v18k = tuple(
    str(x)
    for x in FD004_DRIFT_FULL_MC_MATRIX_SHA256_630V18J
)

scenario_evidence_630v18k = tuple(
    FD004_DRIFT_SCENARIO_EVIDENCE_630V18J
)


shape_checks_630v18k = {
    "prediction":
        pred_630v18k.shape == (102, 10824),

    "mc_std":
        std_630v18k.shape == (102, 10824),

    "final_risk":
        risk_630v18k.shape == (102, 10824),
}

validity_checks_630v18k = {
    "prediction_finite":
        bool(np.isfinite(pred_630v18k).all()),

    "mc_std_finite":
        bool(np.isfinite(std_630v18k).all()),

    "mc_std_nonnegative":
        bool(np.all(std_630v18k >= 0.0)),

    "risk_finite":
        bool(np.isfinite(risk_630v18k).all()),

    "risk_in_unit_interval":
        bool(
            np.all(
                (risk_630v18k >= 0.0)
                &
                (risk_630v18k <= 1.0)
            )
        ),
}

if not all(shape_checks_630v18k.values()):
    raise RuntimeError(
        "Drift retained-array shape changed."
    )

if not all(validity_checks_630v18k.values()):
    raise RuntimeError(
        "Drift retained-array validity check failed."
    )


# =============================================================================
# 3) RESIDENT AUTHORITATIVE HASHES — NO MANUAL VALUES
# =============================================================================

resident_authority_hashes_630v18k = {
    "prediction":
        str(
            FD004_DRIFT_BLOCK_PREDICTIONS_SHA256_630V18J
        ),

    "mc_std":
        str(
            FD004_DRIFT_BLOCK_MC_STD_SHA256_630V18J
        ),

    "final_risk":
        str(
            FD004_DRIFT_BLOCK_FINAL_RISK_SHA256_630V18J
        ),

    "block_contract":
        str(
            FD004_DRIFT_BLOCK_CONTRACT_SHA256_630V18J
        ),

    "checkpoint_npz":
        str(
            FD004_DRIFT_CHECKPOINT_NPZ_SHA256_630V18J
        ),

    "checkpoint_json":
        str(
            FD004_DRIFT_CHECKPOINT_JSON_SHA256_630V18J
        ),
}

for key_630v18k, value_630v18k in (
    resident_authority_hashes_630v18k.items()
):
    if (
        len(value_630v18k) != 64
        or
        not all(
            char in "0123456789abcdef"
            for char in value_630v18k.lower()
        )
    ):
        raise RuntimeError(
            f"Resident {key_630v18k} authority "
            "is not a valid SHA-256."
        )


rehash_checks_630v18k = {
    "prediction":
        str(
            sha256_array_630v2(
                pred_630v18k
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "prediction"
        ],

    "mc_std":
        str(
            sha256_array_630v2(
                std_630v18k
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "mc_std"
        ],

    "final_risk":
        str(
            sha256_array_630v2(
                risk_630v18k
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "final_risk"
        ],
}

if not all(rehash_checks_630v18k.values()):
    failed_630v18k = [
        key
        for key, passed
        in rehash_checks_630v18k.items()
        if not passed
    ]

    raise RuntimeError(
        "Resident drift scientific rehash failed: "
        + ", ".join(failed_630v18k)
    )


# =============================================================================
# 4) MANIFEST / SCENARIO EVIDENCE
# =============================================================================

manifest_checks_630v18k = {
    "row_count":
        len(manifest_630v18k) == 102,

    "family_drift":
        bool(
            np.all(
                manifest_630v18k[
                    "family"
                ].astype(str).to_numpy()
                ==
                "drift"
            )
        ),

    "number_min":
        int(
            manifest_630v18k[
                "scenario_number"
            ].min()
        )
        == 196,

    "number_max":
        int(
            manifest_630v18k[
                "scenario_number"
            ].max()
        )
        == 315,

    "numbers_strictly_increasing":
        bool(
            np.all(
                np.diff(
                    manifest_630v18k[
                        "scenario_number"
                    ].to_numpy(
                        dtype=np.int64
                    )
                )
                > 0
            )
        ),

    "alpha_complete":
        bool(
            manifest_630v18k[
                "alpha"
            ].notna().all()
        ),

    "direction_complete":
        bool(
            manifest_630v18k[
                "direction"
            ].notna().all()
        ),

    "direction_valid":
        bool(
            manifest_630v18k[
                "direction"
            ]
            .astype(float)
            .isin([-1.0, 1.0])
            .all()
        ),
}

if not all(manifest_checks_630v18k.values()):
    raise RuntimeError(
        "Drift manifest closure check failed."
    )


if len(scenario_evidence_630v18k) != 102:
    raise RuntimeError(
        "Drift scenario-evidence count changed."
    )

evidence_ids_630v18k = np.asarray(
    [
        str(row["scenario_id"])
        for row
        in scenario_evidence_630v18k
    ],
    dtype=object,
)

evidence_numbers_630v18k = np.asarray(
    [
        int(row["scenario_number"])
        for row
        in scenario_evidence_630v18k
    ],
    dtype=np.int64,
)

evidence_order_exact_630v18k = bool(
    np.array_equal(
        evidence_ids_630v18k,
        manifest_630v18k[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
    and
    np.array_equal(
        evidence_numbers_630v18k,
        manifest_630v18k[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
    )
)

if not evidence_order_exact_630v18k:
    raise RuntimeError(
        "Drift scenario-evidence order changed."
    )


mc_hash_checks_630v18k = {
    "count_102":
        len(mc_hashes_630v18k) == 102,

    "format_exact":
        bool(
            all(
                len(value) == 64
                and
                all(
                    char in "0123456789abcdef"
                    for char in value.lower()
                )
                for value in mc_hashes_630v18k
            )
        ),
}

if not all(mc_hash_checks_630v18k.values()):
    raise RuntimeError(
        "Drift full-MC hash evidence changed."
    )


# =============================================================================
# 5) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18k = dict(
    FD004_DRIFT_BLOCK_CONTRACT_630V18J
)

contract_recomputed_sha_630v18k = (
    hashlib.sha256(
        json.dumps(
            contract_630v18k,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

contract_checks_630v18k = {
    "sha_exact":
        contract_recomputed_sha_630v18k
        ==
        resident_authority_hashes_630v18k[
            "block_contract"
        ],

    "family":
        str(
            contract_630v18k.get(
                "family"
            )
        )
        ==
        "drift",

    "scenario_count":
        int(
            contract_630v18k.get(
                "scenario_count"
            )
        )
        ==
        102,

    "prediction_hash":
        str(
            contract_630v18k.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "prediction"
        ],

    "mc_std_hash":
        str(
            contract_630v18k.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "mc_std"
        ],

    "risk_hash":
        str(
            contract_630v18k.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "final_risk"
        ],

    "noise_closure_link":
        str(
            contract_630v18k.get(
                "noise_final_closure_sha256"
            )
        )
        ==
        str(
            FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1
        ),

    "full_mc_not_retained":
        bool(
            contract_630v18k.get(
                "full_mc_matrices_retained"
            )
        )
        is False,

    "base_unchanged":
        bool(
            contract_630v18k.get(
                "corruption_base_unchanged"
            )
        )
        is True,

    "model_state_unchanged":
        bool(
            contract_630v18k.get(
                "model_state_unchanged"
            )
        )
        is True,

    "model_mode_unchanged":
        bool(
            contract_630v18k.get(
                "model_mode_unchanged"
            )
        )
        is True,

    "dev_outcomes_unused":
        bool(
            contract_630v18k.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18k.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            contract_630v18k.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18k.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18k.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(contract_checks_630v18k.values()):
    failed_630v18k = [
        key
        for key, passed
        in contract_checks_630v18k.items()
        if not passed
    ]

    raise RuntimeError(
        "Drift block-contract closure failed: "
        + ", ".join(failed_630v18k)
    )


# =============================================================================
# 6) CHECKPOINT BYTE INTEGRITY
# =============================================================================

prior_roundtrip_checks_630v18k = dict(
    FD004_DRIFT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18J
)

prior_roundtrip_all_true_630v18k = bool(
    prior_roundtrip_checks_630v18k
    and
    all(
        bool(value)
        for value in prior_roundtrip_checks_630v18k.values()
    )
)

if not prior_roundtrip_all_true_630v18k:
    raise RuntimeError(
        "Prior drift checkpoint round-trip evidence changed."
    )


npz_path_630v18k = Path(
    FD004_DRIFT_CHECKPOINT_NPZ_PATH_630V18J
)

json_path_630v18k = Path(
    FD004_DRIFT_CHECKPOINT_JSON_PATH_630V18J
)

if not npz_path_630v18k.is_file():
    raise RuntimeError(
        "Drift NPZ checkpoint file is missing."
    )

if not json_path_630v18k.is_file():
    raise RuntimeError(
        "Drift JSON checkpoint file is missing."
    )


def _file_sha256_630v18k(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


checkpoint_byte_checks_630v18k = {
    "npz":
        _file_sha256_630v18k(
            npz_path_630v18k
        )
        ==
        resident_authority_hashes_630v18k[
            "checkpoint_npz"
        ],

    "json":
        _file_sha256_630v18k(
            json_path_630v18k
        )
        ==
        resident_authority_hashes_630v18k[
            "checkpoint_json"
        ],
}

if not all(
    checkpoint_byte_checks_630v18k.values()
):
    raise RuntimeError(
        "Drift checkpoint byte hash changed."
    )


# =============================================================================
# 7) SAFE NPZ REOPEN — allow_pickle=False
# =============================================================================

expected_npz_keys_630v18k = tuple(
    sorted(
        (
            "scenario_number",
            "scenario_id",
            "deterministic_prediction",
            "mc_std",
            "final_selective_risk",
            "full_mc_matrix_sha256",
        )
    )
)

with np.load(
    npz_path_630v18k,
    allow_pickle=False,
) as npz_630v18k:

    keys_630v18k = tuple(
        sorted(
            npz_630v18k.files
        )
    )

    if (
        keys_630v18k
        !=
        expected_npz_keys_630v18k
    ):
        raise RuntimeError(
            "Drift NPZ key set changed."
        )

    loaded_numbers_630v18k = np.asarray(
        npz_630v18k[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    loaded_id_raw_630v18k = np.asarray(
        npz_630v18k[
            "scenario_id"
        ]
    )

    loaded_ids_630v18k = (
        loaded_id_raw_630v18k
        .astype(str)
        .reshape(-1)
    )

    loaded_pred_630v18k = np.asarray(
        npz_630v18k[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    loaded_std_630v18k = np.asarray(
        npz_630v18k[
            "mc_std"
        ],
        dtype=np.float64,
    )

    loaded_risk_630v18k = np.asarray(
        npz_630v18k[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    loaded_mc_hashes_630v18k = tuple(
        str(x)
        for x in np.asarray(
            npz_630v18k[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


npz_checks_630v18k = {
    "scenario_id_non_object":
        bool(
            not loaded_id_raw_630v18k.dtype.hasobject
        ),

    "scenario_number":
        bool(
            np.array_equal(
                loaded_numbers_630v18k,
                manifest_630v18k[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                loaded_ids_630v18k,
                manifest_630v18k[
                    "scenario_id"
                ].astype(str).to_numpy(),
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                loaded_pred_630v18k,
                pred_630v18k,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                loaded_std_630v18k,
                std_630v18k,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                loaded_risk_630v18k,
                risk_630v18k,
            )
        ),

    "full_mc_hashes":
        loaded_mc_hashes_630v18k
        ==
        mc_hashes_630v18k,
}

if not all(npz_checks_630v18k.values()):
    failed_630v18k = [
        key
        for key, passed
        in npz_checks_630v18k.items()
        if not passed
    ]

    raise RuntimeError(
        "Drift NPZ safe round-trip failed: "
        + ", ".join(failed_630v18k)
    )


# =============================================================================
# 8) JSON CHECKPOINT REVERIFY
# =============================================================================

with open(
    json_path_630v18k,
    "r",
    encoding="utf-8",
) as file:

    json_loaded_630v18k = json.load(
        file
    )


json_checks_630v18k = {
    "dataset":
        str(
            json_loaded_630v18k.get(
                "dataset"
            )
        )
        ==
        "FD004",

    "stage":
        str(
            json_loaded_630v18k.get(
                "stage"
            )
        )
        ==
        "6.30V-18J",

    "family":
        str(
            json_loaded_630v18k.get(
                "family"
            )
        )
        ==
        "drift",

    "scenario_count":
        int(
            json_loaded_630v18k.get(
                "scenario_count"
            )
        )
        ==
        102,

    "npz_sha":
        str(
            json_loaded_630v18k.get(
                "npz_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "checkpoint_npz"
        ],

    "block_contract_sha":
        str(
            json_loaded_630v18k.get(
                "block_contract_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "block_contract"
        ],

    "prediction_sha":
        str(
            json_loaded_630v18k.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "prediction"
        ],

    "mc_std_sha":
        str(
            json_loaded_630v18k.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "mc_std"
        ],

    "risk_sha":
        str(
            json_loaded_630v18k.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_authority_hashes_630v18k[
            "final_risk"
        ],

    "scenario_evidence_count":
        len(
            json_loaded_630v18k.get(
                "scenario_evidence",
                [],
            )
        )
        ==
        102,

    "dev_outcomes_unused":
        bool(
            json_loaded_630v18k.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            json_loaded_630v18k.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            json_loaded_630v18k.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            json_loaded_630v18k.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            json_loaded_630v18k.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(json_checks_630v18k.values()):
    failed_630v18k = [
        key
        for key, passed
        in json_checks_630v18k.items()
        if not passed
    ]

    raise RuntimeError(
        "Drift JSON verification failed: "
        + ", ".join(failed_630v18k)
    )


# =============================================================================
# 9) FINAL DRIFT CLOSURE
# =============================================================================

FD004_DRIFT_FINAL_CLOSURE_630V18K = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18K",

    "family":
        "drift",

    "status":
        "DRIFT_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        102,

    "prediction_block_sha256":
        resident_authority_hashes_630v18k[
            "prediction"
        ],

    "mc_std_block_sha256":
        resident_authority_hashes_630v18k[
            "mc_std"
        ],

    "final_risk_block_sha256":
        resident_authority_hashes_630v18k[
            "final_risk"
        ],

    "drift_block_contract_sha256":
        resident_authority_hashes_630v18k[
            "block_contract"
        ],

    "safe_checkpoint_npz_sha256":
        resident_authority_hashes_630v18k[
            "checkpoint_npz"
        ],

    "safe_checkpoint_json_sha256":
        resident_authority_hashes_630v18k[
            "checkpoint_json"
        ],

    "safe_checkpoint_roundtrip_exact":
        True,

    "full_mc_matrix_hash_count":
        len(
            mc_hashes_630v18k
        ),

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K = (
    hashlib.sha256(
        json.dumps(
            FD004_DRIFT_FINAL_CLOSURE_630V18K,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DRIFT_FAMILY_CLOSED_630V18K = True

FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18K = False
FD004_DEV_TARGETS_ERRORS_USED_630V18K = False
FD004_THRESHOLD_SELECTED_630V18K = False
FD004_POLICY_SELECTED_630V18K = False
FD004_TEST_CONTENTS_READ_630V18K = False
FD004_RUL_CONTENTS_READ_630V18K = False

all_checks_passed_630v18k = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18K — FD004 DRIFT FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Drift scenarios rerun             :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- RESIDENT SCIENTIFIC EVIDENCE ---")

print(
    "Prediction rehash exact           :",
    rehash_checks_630v18k[
        "prediction"
    ],
)

print(
    "MC STD rehash exact               :",
    rehash_checks_630v18k[
        "mc_std"
    ],
)

print(
    "Final-risk rehash exact           :",
    rehash_checks_630v18k[
        "final_risk"
    ],
)

print(
    "Manifest checks all true          :",
    all(
        manifest_checks_630v18k.values()
    ),
)

print(
    "Scenario evidence order exact     :",
    evidence_order_exact_630v18k,
)

print(
    "Full-MC hash evidence exact       :",
    all(
        mc_hash_checks_630v18k.values()
    ),
)


print("\n--- BLOCK CONTRACT ---")

for key, value in contract_checks_630v18k.items():
    print(
        f"{key:28s}:",
        value,
    )


print("\n--- SAFE CHECKPOINT ---")

print(
    "NPZ byte SHA exact                :",
    checkpoint_byte_checks_630v18k[
        "npz"
    ],
)

print(
    "JSON byte SHA exact               :",
    checkpoint_byte_checks_630v18k[
        "json"
    ],
)

print(
    "NPZ allow_pickle=False checks     :",
    npz_checks_630v18k,
)

print(
    "JSON checks                       :",
    json_checks_630v18k,
)

print(
    "Prior 18J round-trip all true     :",
    prior_roundtrip_all_true_630v18k,
)

print(
    "Noise closure link exact          :",
    noise_closure_link_exact_630v18k,
)


print("\n--- FINAL CLOSURE ---")

print(
    "Drift family closed               :",
    FD004_DRIFT_FAMILY_CLOSED_630V18K,
)

print(
    "Drift final closure SHA-256       :",
    FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K,
)


print(
    "\nPASS — FD004 drift is evidence-locked using only resident "
    "authoritative hashes and direct recomputation. No drift "
    "scenario was rerun."
)

print(
    "Next: execute only the 63 admissible dropout scenarios."
)

In [ ]:
# STEP 6.30V-18L — FD004 DROPOUT FAMILY EXECUTION
# 63 admissible dropout scenarios only.
#
# LABEL-FREE:
# corruption -> normalization -> windows -> deterministic prediction
# -> 50-pass MC dropout -> reliability chain
#
# NO DEV targets/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.
#
# Retain:
# - deterministic prediction
# - MC STD
# - final selective risk
# - SHA-256 of each full MC matrix, then discard full MC matrix.
#
# Checkpoint:
# - fixed-width Unicode scenario IDs
# - allow_pickle=False round-trip verification
# - persistence failure does NOT invalidate completed scientific execution.

import numpy as np
import pandas as pd
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18l = (
    "all_checks_passed_630v18k",
    "FD004_DRIFT_FAMILY_CLOSED_630V18K",
    "FD004_DRIFT_FINAL_CLOSURE_630V18K",
    "FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_DETERMINISTIC_PREDICTOR_630V18CR1",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
    "FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0",
)

missing_630v18l = [
    name
    for name in required_630v18l
    if name not in globals()
]

if missing_630v18l:
    raise RuntimeError(
        "Missing STEP 6.30V-18L authority object(s): "
        + ", ".join(missing_630v18l)
    )

if not bool(all_checks_passed_630v18k):
    raise RuntimeError(
        "STEP 6.30V-18K must remain passed."
    )

if not bool(FD004_DRIFT_FAMILY_CLOSED_630V18K):
    raise RuntimeError(
        "Drift family must remain evidence-locked before dropout."
    )

drift_closure_recomputed_sha_630v18l = hashlib.sha256(
    json.dumps(
        FD004_DRIFT_FINAL_CLOSURE_630V18K,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    drift_closure_recomputed_sha_630v18l
    !=
    str(FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K)
):
    raise RuntimeError(
        "Resident drift final-closure evidence changed."
    )


# =============================================================================
# 1) FREEZE DROPOUT BLOCK
# =============================================================================

execution_manifest_630v18l = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

dropout_manifest_630v18l = (
    execution_manifest_630v18l.loc[
        execution_manifest_630v18l[
            "family"
        ].astype(str)
        == "dropout"
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(dropout_manifest_630v18l) != 63:
    raise RuntimeError(
        "Expected exactly 63 admissible dropout scenarios."
    )

dropout_numbers_630v18l = (
    dropout_manifest_630v18l[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

if (
    int(dropout_numbers_630v18l.min()) != 316
    or
    int(dropout_numbers_630v18l.max()) != 378
):
    raise RuntimeError(
        "Frozen dropout scenario-number range changed."
    )

if not np.all(
    dropout_manifest_630v18l[
        "family"
    ].astype(str).to_numpy()
    ==
    "dropout"
):
    raise RuntimeError(
        "Non-dropout row entered the dropout block."
    )

if not (
    dropout_manifest_630v18l[
        "requested_fraction"
    ].notna().all()
    and
    dropout_manifest_630v18l[
        "seed"
    ].notna().all()
):
    raise RuntimeError(
        "Dropout manifest has missing requested_fraction/seed values."
    )

requested_fractions_630v18l = (
    dropout_manifest_630v18l[
        "requested_fraction"
    ].to_numpy(dtype=np.float64)
)

if not np.all(
    (requested_fractions_630v18l > 0.0)
    &
    (requested_fractions_630v18l < 1.0)
):
    raise RuntimeError(
        "Dropout manifest has invalid requested_fraction values."
    )


# =============================================================================
# 2) FROZEN AUTHORITIES
# =============================================================================

base_630v18l = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v18l = (
    base_630v18l.copy(deep=True)
)

if "regime_id" not in base_630v18l.columns:
    raise RuntimeError(
        "Frozen dropout corruption base has no regime_id column."
    )

regime_ids_630v18l = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

if not np.array_equal(
    base_630v18l[
        "regime_id"
    ].to_numpy(dtype=np.int64),
    regime_ids_630v18l,
):
    raise RuntimeError(
        "Frozen dropout corruption-base regime_id vector changed."
    )

expected_window_units_630v18l = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

expected_endpoint_cycles_630v18l = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if expected_window_units_630v18l.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV window-unit authority shape changed."
    )

if expected_endpoint_cycles_630v18l.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle authority shape changed."
    )

model_630v18l = MODEL_FD004_630I
device_630v18l = FD004_DEVICE_630I

if bool(model_630v18l.training):
    raise RuntimeError(
        "FD004 model must be in eval mode before dropout execution."
    )


def _model_state_sha_630v18l(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )
        arr = tensor.numpy()

        h.update(key.encode("utf-8"))
        h.update(str(arr.dtype).encode("utf-8"))
        h.update(str(tuple(arr.shape)).encode("utf-8"))
        h.update(arr.tobytes(order="C"))

    return h.hexdigest()


model_state_before_630v18l = (
    _model_state_sha_630v18l(model_630v18l)
)

model_mode_before_630v18l = bool(
    model_630v18l.training
)


# =============================================================================
# 3) PREALLOCATE RETAINED PRODUCTS
# =============================================================================

n_scenarios_630v18l = 63
n_windows_630v18l = 10824

dropout_pred_matrix_630v18l = np.empty(
    (n_scenarios_630v18l, n_windows_630v18l),
    dtype=np.float32,
)

dropout_mc_std_matrix_630v18l = np.empty(
    (n_scenarios_630v18l, n_windows_630v18l),
    dtype=np.float64,
)

dropout_final_risk_matrix_630v18l = np.empty(
    (n_scenarios_630v18l, n_windows_630v18l),
    dtype=np.float64,
)

dropout_mc_matrix_hashes_630v18l = []
dropout_scenario_evidence_630v18l = []


# =============================================================================
# 4) EXECUTE 63 DROPOUT SCENARIOS
# =============================================================================

print("=" * 176)
print(
    "STEP 6.30V-18L — FD004 DROPOUT FAMILY EXECUTION "
    "(63 ADMISSIBLE SCENARIOS)"
)
print("=" * 176)

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nStarting frozen dropout block      :",
    n_scenarios_630v18l,
    "scenarios",
)


for row_index_630v18l, row_630v18l in (
    dropout_manifest_630v18l.iterrows()
):

    scenario_id_630v18l = str(
        row_630v18l["scenario_id"]
    )

    scenario_number_630v18l = int(
        row_630v18l["scenario_number"]
    )

    sensor_630v18l = str(
        row_630v18l["sensor"]
    )

    requested_fraction_630v18l = float(
        row_630v18l["requested_fraction"]
    )

    seed_630v18l = int(
        row_630v18l["seed"]
    )

    corrupted_raw_630v18l = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v18l,
            row_630v18l,
        )
    )

    if not isinstance(
        corrupted_raw_630v18l,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: dispatcher did not return DataFrame."
        )

    if corrupted_raw_630v18l.shape != (12274, 27):
        raise RuntimeError(
            f"{scenario_id_630v18l}: corrupted raw shape changed."
        )

    if not np.array_equal(
        corrupted_raw_630v18l[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_ids_630v18l,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: regime_id changed under dropout."
        )

    if not np.isfinite(
        corrupted_raw_630v18l[
            sensor_630v18l
        ].to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18l}: dropout output contains non-finite "
            "target-sensor values after frozen imputation."
        )

    normalized_630v18l = (
        normalize_fd004_raw_630v2(
            corrupted_raw_630v18l,
            regime_ids_630v18l,
        )
    )

    (
        X_630v18l,
        unit_ids_630v18l,
        endpoint_cycles_630v18l,
    ) = build_windows_fd004_630v2(
        normalized_630v18l
    )

    X_630v18l = np.asarray(
        X_630v18l,
        dtype=np.float32,
    )

    if X_630v18l.shape != (10824, 30, 24):
        raise RuntimeError(
            f"{scenario_id_630v18l}: window tensor shape changed."
        )

    if not np.isfinite(X_630v18l).all():
        raise RuntimeError(
            f"{scenario_id_630v18l}: non-finite model input."
        )

    if not np.array_equal(
        np.asarray(
            unit_ids_630v18l,
            dtype=np.int64,
        ).reshape(-1),
        expected_window_units_630v18l,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: window-unit membership changed."
        )

    if not np.array_equal(
        np.asarray(
            endpoint_cycles_630v18l,
            dtype=np.int64,
        ).reshape(-1),
        expected_endpoint_cycles_630v18l,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: endpoint-cycle alignment changed."
        )

    deterministic_pred_630v18l = np.asarray(
        FD004_DETERMINISTIC_PREDICTOR_630V18CR1(
            X_630v18l,
            model_630v18l,
            device_630v18l,
            batch_size=512,
        ),
        dtype=np.float32,
    ).reshape(-1)

    if deterministic_pred_630v18l.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18l}: deterministic prediction shape changed."
        )

    if not np.isfinite(
        deterministic_pred_630v18l
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18l}: non-finite deterministic prediction."
        )

    (
        mc_matrix_630v18l,
        mc_mean_630v18l,
        mc_std_630v18l,
    ) = fd004_mc_dropout_predict_630v4r2(
        X_630v18l,
        model_630v18l,
        device_630v18l,
    )

    mc_matrix_630v18l = np.asarray(
        mc_matrix_630v18l,
        dtype=np.float32,
    )

    mc_std_630v18l = np.asarray(
        mc_std_630v18l,
        dtype=np.float64,
    ).reshape(-1)

    if mc_matrix_630v18l.shape != (50, 10824):
        raise RuntimeError(
            f"{scenario_id_630v18l}: MC matrix shape changed."
        )

    if mc_std_630v18l.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18l}: MC STD shape changed."
        )

    if not (
        np.isfinite(mc_matrix_630v18l).all()
        and
        np.isfinite(mc_std_630v18l).all()
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: non-finite MC output."
        )

    if np.any(mc_std_630v18l < 0.0):
        raise RuntimeError(
            f"{scenario_id_630v18l}: negative MC STD."
        )

    mc_matrix_sha_630v18l = str(
        sha256_array_630v2(
            mc_matrix_630v18l
        )
    )

    (
        phase_ids_630v18l,
        adjusted_630v18l,
        g1_630v18l,
        final_risk_630v18l,
    ) = apply_fd004_final_reliability_chain_630v4r2(
        deterministic_pred_630v18l,
        mc_std_630v18l,
    )

    final_risk_630v18l = np.asarray(
        final_risk_630v18l,
        dtype=np.float64,
    ).reshape(-1)

    if final_risk_630v18l.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18l}: final risk shape changed."
        )

    if not np.isfinite(
        final_risk_630v18l
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18l}: non-finite final risk."
        )

    if np.any(
        (final_risk_630v18l < 0.0)
        |
        (final_risk_630v18l > 1.0)
    ):
        raise RuntimeError(
            f"{scenario_id_630v18l}: final risk escaped [0,1]."
        )

    dropout_pred_matrix_630v18l[
        row_index_630v18l
    ] = deterministic_pred_630v18l

    dropout_mc_std_matrix_630v18l[
        row_index_630v18l
    ] = mc_std_630v18l

    dropout_final_risk_matrix_630v18l[
        row_index_630v18l
    ] = final_risk_630v18l

    dropout_mc_matrix_hashes_630v18l.append(
        mc_matrix_sha_630v18l
    )

    dropout_scenario_evidence_630v18l.append(
        {
            "scenario_id":
                scenario_id_630v18l,

            "scenario_number":
                scenario_number_630v18l,

            "family":
                "dropout",

            "sensor":
                sensor_630v18l,

            "requested_fraction":
                requested_fraction_630v18l,

            "seed":
                seed_630v18l,

            "deterministic_prediction_sha256":
                str(
                    sha256_array_630v2(
                        deterministic_pred_630v18l
                    )
                ),

            "mc_matrix_sha256":
                mc_matrix_sha_630v18l,

            "mc_std_sha256":
                str(
                    sha256_array_630v2(
                        mc_std_630v18l
                    )
                ),

            "final_selective_risk_sha256":
                str(
                    sha256_array_630v2(
                        final_risk_630v18l
                    )
                ),
        }
    )

    del mc_matrix_630v18l
    del mc_mean_630v18l
    del phase_ids_630v18l
    del adjusted_630v18l
    del g1_630v18l
    del corrupted_raw_630v18l
    del normalized_630v18l
    del X_630v18l

    if (
        (row_index_630v18l + 1) == 1
        or
        (row_index_630v18l + 1) % 10 == 0
        or
        (row_index_630v18l + 1) == n_scenarios_630v18l
    ):
        print(
            f"Completed dropout scenario "
            f"{row_index_630v18l + 1:2d}/"
            f"{n_scenarios_630v18l}: "
            f"{scenario_id_630v18l}"
        )


# =============================================================================
# 5) POST-BLOCK SAFETY
# =============================================================================

if not np.isfinite(
    dropout_pred_matrix_630v18l
).all():
    raise RuntimeError(
        "Retained dropout deterministic predictions contain non-finite values."
    )

if not np.isfinite(
    dropout_mc_std_matrix_630v18l
).all():
    raise RuntimeError(
        "Retained dropout MC STD contains non-finite values."
    )

if not np.isfinite(
    dropout_final_risk_matrix_630v18l
).all():
    raise RuntimeError(
        "Retained dropout final risk contains non-finite values."
    )

if np.any(
    dropout_mc_std_matrix_630v18l < 0.0
):
    raise RuntimeError(
        "Retained dropout MC STD contains negative values."
    )

if np.any(
    (dropout_final_risk_matrix_630v18l < 0.0)
    |
    (dropout_final_risk_matrix_630v18l > 1.0)
):
    raise RuntimeError(
        "Retained dropout risk escaped [0,1]."
    )

if len(
    dropout_mc_matrix_hashes_630v18l
) != 63:
    raise RuntimeError(
        "Dropout MC matrix hash count changed."
    )

if len(
    dropout_scenario_evidence_630v18l
) != 63:
    raise RuntimeError(
        "Dropout scenario evidence count changed."
    )

base_unchanged_630v18l = bool(
    base_630v18l.equals(
        base_snapshot_630v18l
    )
)

if not base_unchanged_630v18l:
    raise RuntimeError(
        "Frozen corruption base changed during dropout execution."
    )

model_state_after_630v18l = (
    _model_state_sha_630v18l(
        model_630v18l
    )
)

model_mode_after_630v18l = bool(
    model_630v18l.training
)

model_state_unchanged_630v18l = bool(
    model_state_before_630v18l
    ==
    model_state_after_630v18l
)

model_mode_unchanged_630v18l = bool(
    model_mode_before_630v18l
    ==
    model_mode_after_630v18l
)

if not model_state_unchanged_630v18l:
    raise RuntimeError(
        "Model state changed during dropout block."
    )

if not model_mode_unchanged_630v18l:
    raise RuntimeError(
        "Model mode changed during dropout block."
    )


# =============================================================================
# 6) FREEZE DROPOUT EVIDENCE
# =============================================================================

FD004_DROPOUT_SCENARIO_MANIFEST_630V18L = (
    dropout_manifest_630v18l.copy()
)

FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L = (
    dropout_pred_matrix_630v18l
)

FD004_DROPOUT_MC_STD_630V18L = (
    dropout_mc_std_matrix_630v18l
)

FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L = (
    dropout_final_risk_matrix_630v18l
)

FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L = tuple(
    dropout_mc_matrix_hashes_630v18l
)

FD004_DROPOUT_SCENARIO_EVIDENCE_630V18L = tuple(
    dropout_scenario_evidence_630v18l
)

FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L = str(
    sha256_array_630v2(
        FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L
    )
)

FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L = str(
    sha256_array_630v2(
        FD004_DROPOUT_MC_STD_630V18L
    )
)

FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L = str(
    sha256_array_630v2(
        FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L
    )
)

dropout_block_contract_630v18l = {
    "dataset": "FD004",
    "stage": "6.30V-18L",
    "family": "dropout",
    "status": "DROPOUT_BLOCK_COMPLETE_LABEL_FREE",
    "scenario_count": 63,

    "scenario_number_min":
        int(dropout_numbers_630v18l.min()),

    "scenario_number_max":
        int(dropout_numbers_630v18l.max()),

    "deterministic_prediction_shape":
        tuple(
            FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L.shape
        ),

    "mc_std_shape":
        tuple(
            FD004_DROPOUT_MC_STD_630V18L.shape
        ),

    "final_selective_risk_shape":
        tuple(
            FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L.shape
        ),

    "full_mc_matrix_hash_count":
        len(
            FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L
        ),

    "full_mc_matrices_retained": False,

    "prediction_block_sha256":
        FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L,

    "mc_std_block_sha256":
        FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L,

    "final_risk_block_sha256":
        FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L,

    "drift_final_closure_sha256":
        str(
            FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K
        ),

    "admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helper_return_audit_sha256":
        str(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0
        ),

    "corruption_base_unchanged":
        base_unchanged_630v18l,

    "model_state_unchanged":
        model_state_unchanged_630v18l,

    "model_mode_unchanged":
        model_mode_unchanged_630v18l,

    "dev_targets_errors_used": False,
    "threshold_selected": False,
    "policy_selected": False,
    "test_contents_read": False,
    "official_rul_contents_read": False,
}

FD004_DROPOUT_BLOCK_CONTRACT_630V18L = (
    dropout_block_contract_630v18l
)

FD004_DROPOUT_BLOCK_CONTRACT_SHA256_630V18L = (
    hashlib.sha256(
        json.dumps(
            dropout_block_contract_630v18l,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) SAFE CHECKPOINT
# =============================================================================

checkpoint_saved_630v18l = False
checkpoint_roundtrip_exact_630v18l = False
checkpoint_error_630v18l = None

checkpoint_npz_path_630v18l = None
checkpoint_json_path_630v18l = None
checkpoint_npz_sha_630v18l = None
checkpoint_json_sha_630v18l = None
checkpoint_roundtrip_checks_630v18l = None

try:
    checkpoint_dir_630v18l = Path(
        r"C:\RUL_Reliability_Research\checkpoints"
    )

    checkpoint_dir_630v18l.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_npz_path_630v18l = (
        checkpoint_dir_630v18l
        /
        "fd004_dropout_block_630v18l_v1_safe.npz"
    )

    checkpoint_json_path_630v18l = (
        checkpoint_dir_630v18l
        /
        "fd004_dropout_block_630v18l_v1_safe.json"
    )

    npz_tmp_630v18l = (
        checkpoint_dir_630v18l
        /
        "fd004_dropout_block_630v18l_v1_safe.npz.tmp"
    )

    json_tmp_630v18l = (
        checkpoint_dir_630v18l
        /
        "fd004_dropout_block_630v18l_v1_safe.json.tmp"
    )

    scenario_id_list_630v18l = (
        dropout_manifest_630v18l[
            "scenario_id"
        ]
        .astype(str)
        .tolist()
    )

    scenario_id_max_len_630v18l = max(
        len(value)
        for value in scenario_id_list_630v18l
    )

    scenario_id_unicode_630v18l = np.asarray(
        scenario_id_list_630v18l,
        dtype=f"<U{scenario_id_max_len_630v18l}",
    )

    full_mc_hash_unicode_630v18l = np.asarray(
        FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L,
        dtype="<U64",
    )

    with open(
        npz_tmp_630v18l,
        "wb",
    ) as file:
        np.savez_compressed(
            file,

            scenario_number=
                dropout_manifest_630v18l[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),

            scenario_id=
                scenario_id_unicode_630v18l,

            deterministic_prediction=
                FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L,

            mc_std=
                FD004_DROPOUT_MC_STD_630V18L,

            final_selective_risk=
                FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L,

            full_mc_matrix_sha256=
                full_mc_hash_unicode_630v18l,
        )

    os.replace(
        npz_tmp_630v18l,
        checkpoint_npz_path_630v18l,
    )

    def _file_sha256_630v18l(path):
        h = hashlib.sha256()

        with open(path, "rb") as file:
            while True:
                chunk = file.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                h.update(chunk)

        return h.hexdigest()

    checkpoint_npz_sha_630v18l = (
        _file_sha256_630v18l(
            checkpoint_npz_path_630v18l
        )
    )

    checkpoint_metadata_630v18l = {
        "dataset": "FD004",
        "stage": "6.30V-18L",
        "family": "dropout",
        "status": "SAFE_SERIALIZATION_CHECKPOINT",
        "scenario_count": 63,

        "scenario_id_dtype":
            str(scenario_id_unicode_630v18l.dtype),

        "block_contract_sha256":
            FD004_DROPOUT_BLOCK_CONTRACT_SHA256_630V18L,

        "prediction_block_sha256":
            FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L,

        "mc_std_block_sha256":
            FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L,

        "final_risk_block_sha256":
            FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L,

        "npz_sha256":
            checkpoint_npz_sha_630v18l,

        "scenario_evidence":
            list(
                FD004_DROPOUT_SCENARIO_EVIDENCE_630V18L
            ),

        "dev_targets_errors_used": False,
        "threshold_selected": False,
        "policy_selected": False,
        "test_contents_read": False,
        "official_rul_contents_read": False,
    }

    with open(
        json_tmp_630v18l,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            checkpoint_metadata_630v18l,
            file,
            sort_keys=True,
            indent=2,
            default=str,
        )

    os.replace(
        json_tmp_630v18l,
        checkpoint_json_path_630v18l,
    )

    checkpoint_json_sha_630v18l = (
        _file_sha256_630v18l(
            checkpoint_json_path_630v18l
        )
    )

    with np.load(
        checkpoint_npz_path_630v18l,
        allow_pickle=False,
    ) as checkpoint_npz_630v18l:

        loaded_numbers_630v18l = np.asarray(
            checkpoint_npz_630v18l[
                "scenario_number"
            ],
            dtype=np.int64,
        ).reshape(-1)

        loaded_id_raw_630v18l = np.asarray(
            checkpoint_npz_630v18l[
                "scenario_id"
            ]
        )

        loaded_ids_630v18l = (
            loaded_id_raw_630v18l
            .astype(str)
            .reshape(-1)
        )

        loaded_pred_630v18l = np.asarray(
            checkpoint_npz_630v18l[
                "deterministic_prediction"
            ],
            dtype=np.float32,
        )

        loaded_std_630v18l = np.asarray(
            checkpoint_npz_630v18l[
                "mc_std"
            ],
            dtype=np.float64,
        )

        loaded_risk_630v18l = np.asarray(
            checkpoint_npz_630v18l[
                "final_selective_risk"
            ],
            dtype=np.float64,
        )

        loaded_mc_hashes_630v18l = tuple(
            str(x)
            for x in np.asarray(
                checkpoint_npz_630v18l[
                    "full_mc_matrix_sha256"
                ]
            ).reshape(-1)
        )

    checkpoint_roundtrip_checks_630v18l = {
        "scenario_id_non_object":
            bool(
                not loaded_id_raw_630v18l.dtype.hasobject
            ),

        "scenario_number":
            bool(
                np.array_equal(
                    loaded_numbers_630v18l,
                    dropout_manifest_630v18l[
                        "scenario_number"
                    ].to_numpy(dtype=np.int64),
                )
            ),

        "scenario_id":
            bool(
                np.array_equal(
                    loaded_ids_630v18l,
                    scenario_id_unicode_630v18l,
                )
            ),

        "prediction":
            bool(
                np.array_equal(
                    loaded_pred_630v18l,
                    FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L,
                )
            ),

        "mc_std":
            bool(
                np.array_equal(
                    loaded_std_630v18l,
                    FD004_DROPOUT_MC_STD_630V18L,
                )
            ),

        "final_risk":
            bool(
                np.array_equal(
                    loaded_risk_630v18l,
                    FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L,
                )
            ),

        "full_mc_hashes":
            loaded_mc_hashes_630v18l
            ==
            tuple(
                FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L
            ),
    }

    checkpoint_roundtrip_exact_630v18l = bool(
        all(
            checkpoint_roundtrip_checks_630v18l.values()
        )
    )

    if not checkpoint_roundtrip_exact_630v18l:
        raise RuntimeError(
            "Dropout safe checkpoint round-trip verification failed."
        )

    checkpoint_saved_630v18l = True

except Exception as exc_630v18l:
    checkpoint_error_630v18l = repr(
        exc_630v18l
    )


# =============================================================================
# 8) FREEZE CHECKPOINT STATUS
# =============================================================================

FD004_DROPOUT_CHECKPOINT_SAVED_630V18L = bool(
    checkpoint_saved_630v18l
)

FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_EXACT_630V18L = bool(
    checkpoint_roundtrip_exact_630v18l
)

FD004_DROPOUT_CHECKPOINT_NPZ_PATH_630V18L = (
    None
    if checkpoint_npz_path_630v18l is None
    else str(checkpoint_npz_path_630v18l)
)

FD004_DROPOUT_CHECKPOINT_JSON_PATH_630V18L = (
    None
    if checkpoint_json_path_630v18l is None
    else str(checkpoint_json_path_630v18l)
)

FD004_DROPOUT_CHECKPOINT_NPZ_SHA256_630V18L = (
    checkpoint_npz_sha_630v18l
)

FD004_DROPOUT_CHECKPOINT_JSON_SHA256_630V18L = (
    checkpoint_json_sha_630v18l
)

FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18L = (
    checkpoint_roundtrip_checks_630v18l
)

FD004_DROPOUT_CHECKPOINT_ERROR_630V18L = (
    checkpoint_error_630v18l
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD004_DROPOUT_BLOCK_COMPLETE_630V18L = True
FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L = 63

FD004_DEV_TARGETS_ERRORS_USED_630V18L = False
FD004_THRESHOLD_SELECTED_630V18L = False
FD004_POLICY_SELECTED_630V18L = False
FD004_TEST_CONTENTS_READ_630V18L = False
FD004_RUL_CONTENTS_READ_630V18L = False

all_checks_passed_630v18l = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("\n--- DROPOUT BLOCK COMPLETE ---")

print(
    "Dropout scenarios executed        :",
    FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L,
)

print(
    "Deterministic retained shape      :",
    FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L.shape,
)

print(
    "MC STD retained shape             :",
    FD004_DROPOUT_MC_STD_630V18L.shape,
)

print(
    "Final risk retained shape         :",
    FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L.shape,
)

print(
    "Full MC matrix hashes retained    :",
    len(
        FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L
    ),
)

print(
    "Full MC matrices retained         :",
    False,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18l,
)

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18l,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18l,
)

print(
    "Prediction block SHA-256          :",
    FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L,
)

print(
    "MC STD block SHA-256              :",
    FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L,
)

print(
    "Final risk block SHA-256          :",
    FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L,
)

print(
    "Dropout block contract SHA-256    :",
    FD004_DROPOUT_BLOCK_CONTRACT_SHA256_630V18L,
)


print("\n--- LABEL-FREE BLOCK SUMMARY ---")

print(
    "MC STD min / median / mean / max  :",
    (
        float(np.min(FD004_DROPOUT_MC_STD_630V18L)),
        float(np.median(FD004_DROPOUT_MC_STD_630V18L)),
        float(np.mean(FD004_DROPOUT_MC_STD_630V18L)),
        float(np.max(FD004_DROPOUT_MC_STD_630V18L)),
    ),
)

print(
    "Risk min / median / mean / max    :",
    (
        float(np.min(FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L)),
        float(np.median(FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L)),
        float(np.mean(FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L)),
        float(np.max(FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L)),
    ),
)


print("\n--- SAFE CHECKPOINT ---")

print(
    "Checkpoint saved                  :",
    FD004_DROPOUT_CHECKPOINT_SAVED_630V18L,
)

print(
    "allow_pickle=False round-trip     :",
    FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_EXACT_630V18L,
)

print(
    "NPZ path                          :",
    FD004_DROPOUT_CHECKPOINT_NPZ_PATH_630V18L,
)

print(
    "JSON path                         :",
    FD004_DROPOUT_CHECKPOINT_JSON_PATH_630V18L,
)

print(
    "NPZ SHA-256                       :",
    FD004_DROPOUT_CHECKPOINT_NPZ_SHA256_630V18L,
)

print(
    "JSON SHA-256                      :",
    FD004_DROPOUT_CHECKPOINT_JSON_SHA256_630V18L,
)

print(
    "Round-trip checks                 :",
    FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18L,
)

print(
    "Checkpoint error                  :",
    FD004_DROPOUT_CHECKPOINT_ERROR_630V18L,
)


print("\n--- LEAKAGE / POLICY GUARDS ---")

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nPASS — the 63 admissible FD004 dropout scenarios are complete "
    "under the frozen label-free execution contract."
)

if (
    checkpoint_saved_630v18l
    and checkpoint_roundtrip_exact_630v18l
):
    print(
        "The dropout block also has a verified pickle-free checkpoint. "
        "Next: close dropout evidence, then execute the final stuck family."
    )
else:
    print(
        "The scientific dropout block is complete, but checkpoint "
        "persistence/verification failed. Do NOT rerun dropout; "
        "repair persistence only."
    )

In [ ]:
# STEP 6.30V-18M — FD004 DROPOUT FAMILY FINAL EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.
#
# IMPORTANT:
# No manually transcribed dropout-result hashes are used.

import numpy as np
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18m = (
    "all_checks_passed_630v18l",
    "FD004_DROPOUT_BLOCK_COMPLETE_630V18L",
    "FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L",
    "FD004_DROPOUT_SCENARIO_MANIFEST_630V18L",
    "FD004_DROPOUT_SCENARIO_EVIDENCE_630V18L",
    "FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L",
    "FD004_DROPOUT_MC_STD_630V18L",
    "FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L",
    "FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L",
    "FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L",
    "FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L",
    "FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L",
    "FD004_DROPOUT_BLOCK_CONTRACT_630V18L",
    "FD004_DROPOUT_BLOCK_CONTRACT_SHA256_630V18L",
    "FD004_DROPOUT_CHECKPOINT_SAVED_630V18L",
    "FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_EXACT_630V18L",
    "FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18L",
    "FD004_DROPOUT_CHECKPOINT_NPZ_PATH_630V18L",
    "FD004_DROPOUT_CHECKPOINT_JSON_PATH_630V18L",
    "FD004_DROPOUT_CHECKPOINT_NPZ_SHA256_630V18L",
    "FD004_DROPOUT_CHECKPOINT_JSON_SHA256_630V18L",
    "FD004_DRIFT_FINAL_CLOSURE_630V18K",
    "FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K",
    "sha256_array_630v2",
)

missing_630v18m = [
    name
    for name in required_630v18m
    if name not in globals()
]

if missing_630v18m:
    raise RuntimeError(
        "Missing STEP 6.30V-18M authority object(s): "
        + ", ".join(missing_630v18m)
    )

if not bool(all_checks_passed_630v18l):
    raise RuntimeError(
        "STEP 6.30V-18L must remain passed."
    )

if not bool(FD004_DROPOUT_BLOCK_COMPLETE_630V18L):
    raise RuntimeError(
        "Dropout block is not marked complete."
    )

if int(FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L) != 63:
    raise RuntimeError(
        "Dropout executed-scenario count changed."
    )

if not bool(FD004_DROPOUT_CHECKPOINT_SAVED_630V18L):
    raise RuntimeError(
        "Dropout checkpoint is not marked saved."
    )

if not bool(FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_EXACT_630V18L):
    raise RuntimeError(
        "Dropout checkpoint is not marked round-trip exact."
    )


# =============================================================================
# 1) VERIFY PRIOR DRIFT CLOSURE LINK
# =============================================================================

drift_closure_recomputed_sha_630v18m = hashlib.sha256(
    json.dumps(
        FD004_DRIFT_FINAL_CLOSURE_630V18K,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

drift_closure_link_exact_630v18m = bool(
    drift_closure_recomputed_sha_630v18m
    ==
    str(FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K)
)

if not drift_closure_link_exact_630v18m:
    raise RuntimeError(
        "Resident drift final-closure evidence changed."
    )


# =============================================================================
# 2) RESIDENT SCIENTIFIC EVIDENCE
# =============================================================================

manifest_630v18m = (
    FD004_DROPOUT_SCENARIO_MANIFEST_630V18L
    .copy()
    .reset_index(drop=True)
)

pred_630v18m = np.asarray(
    FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L,
    dtype=np.float32,
)

std_630v18m = np.asarray(
    FD004_DROPOUT_MC_STD_630V18L,
    dtype=np.float64,
)

risk_630v18m = np.asarray(
    FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L,
    dtype=np.float64,
)

mc_hashes_630v18m = tuple(
    str(x)
    for x in FD004_DROPOUT_FULL_MC_MATRIX_SHA256_630V18L
)

scenario_evidence_630v18m = tuple(
    FD004_DROPOUT_SCENARIO_EVIDENCE_630V18L
)


shape_checks_630v18m = {
    "prediction":
        pred_630v18m.shape == (63, 10824),

    "mc_std":
        std_630v18m.shape == (63, 10824),

    "final_risk":
        risk_630v18m.shape == (63, 10824),
}

validity_checks_630v18m = {
    "prediction_finite":
        bool(np.isfinite(pred_630v18m).all()),

    "mc_std_finite":
        bool(np.isfinite(std_630v18m).all()),

    "mc_std_nonnegative":
        bool(np.all(std_630v18m >= 0.0)),

    "risk_finite":
        bool(np.isfinite(risk_630v18m).all()),

    "risk_in_unit_interval":
        bool(
            np.all(
                (risk_630v18m >= 0.0)
                &
                (risk_630v18m <= 1.0)
            )
        ),
}

if not all(shape_checks_630v18m.values()):
    raise RuntimeError(
        "Dropout retained-array shape changed."
    )

if not all(validity_checks_630v18m.values()):
    raise RuntimeError(
        "Dropout retained-array validity check failed."
    )


# =============================================================================
# 3) RESIDENT AUTHORITATIVE HASHES — NO MANUAL VALUES
# =============================================================================

resident_hashes_630v18m = {
    "prediction":
        str(FD004_DROPOUT_BLOCK_PREDICTIONS_SHA256_630V18L),

    "mc_std":
        str(FD004_DROPOUT_BLOCK_MC_STD_SHA256_630V18L),

    "final_risk":
        str(FD004_DROPOUT_BLOCK_FINAL_RISK_SHA256_630V18L),

    "block_contract":
        str(FD004_DROPOUT_BLOCK_CONTRACT_SHA256_630V18L),

    "checkpoint_npz":
        str(FD004_DROPOUT_CHECKPOINT_NPZ_SHA256_630V18L),

    "checkpoint_json":
        str(FD004_DROPOUT_CHECKPOINT_JSON_SHA256_630V18L),
}

for key_630v18m, value_630v18m in resident_hashes_630v18m.items():
    if (
        len(value_630v18m) != 64
        or
        not all(
            char in "0123456789abcdef"
            for char in value_630v18m.lower()
        )
    ):
        raise RuntimeError(
            f"Resident {key_630v18m} authority is not a valid SHA-256."
        )


rehash_checks_630v18m = {
    "prediction":
        str(
            sha256_array_630v2(
                pred_630v18m
            )
        )
        ==
        resident_hashes_630v18m["prediction"],

    "mc_std":
        str(
            sha256_array_630v2(
                std_630v18m
            )
        )
        ==
        resident_hashes_630v18m["mc_std"],

    "final_risk":
        str(
            sha256_array_630v2(
                risk_630v18m
            )
        )
        ==
        resident_hashes_630v18m["final_risk"],
}

if not all(rehash_checks_630v18m.values()):
    failed_630v18m = [
        key
        for key, passed
        in rehash_checks_630v18m.items()
        if not passed
    ]

    raise RuntimeError(
        "Resident dropout scientific rehash failed: "
        + ", ".join(failed_630v18m)
    )


# =============================================================================
# 4) MANIFEST / SCENARIO EVIDENCE
# =============================================================================

manifest_checks_630v18m = {
    "row_count":
        len(manifest_630v18m) == 63,

    "family_dropout":
        bool(
            np.all(
                manifest_630v18m[
                    "family"
                ].astype(str).to_numpy()
                ==
                "dropout"
            )
        ),

    "number_min":
        int(
            manifest_630v18m[
                "scenario_number"
            ].min()
        )
        == 316,

    "number_max":
        int(
            manifest_630v18m[
                "scenario_number"
            ].max()
        )
        == 378,

    "numbers_strictly_increasing":
        bool(
            np.all(
                np.diff(
                    manifest_630v18m[
                        "scenario_number"
                    ].to_numpy(dtype=np.int64)
                )
                > 0
            )
        ),

    "requested_fraction_complete":
        bool(
            manifest_630v18m[
                "requested_fraction"
            ].notna().all()
        ),

    "seed_complete":
        bool(
            manifest_630v18m[
                "seed"
            ].notna().all()
        ),

    "requested_fraction_valid":
        bool(
            np.all(
                (
                    manifest_630v18m[
                        "requested_fraction"
                    ].to_numpy(dtype=np.float64)
                    > 0.0
                )
                &
                (
                    manifest_630v18m[
                        "requested_fraction"
                    ].to_numpy(dtype=np.float64)
                    < 1.0
                )
            )
        ),
}

if not all(manifest_checks_630v18m.values()):
    raise RuntimeError(
        "Dropout manifest closure check failed."
    )


if len(scenario_evidence_630v18m) != 63:
    raise RuntimeError(
        "Dropout scenario-evidence count changed."
    )

evidence_ids_630v18m = np.asarray(
    [
        str(row["scenario_id"])
        for row in scenario_evidence_630v18m
    ],
    dtype=object,
)

evidence_numbers_630v18m = np.asarray(
    [
        int(row["scenario_number"])
        for row in scenario_evidence_630v18m
    ],
    dtype=np.int64,
)

evidence_order_exact_630v18m = bool(
    np.array_equal(
        evidence_ids_630v18m,
        manifest_630v18m[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
    and
    np.array_equal(
        evidence_numbers_630v18m,
        manifest_630v18m[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
    )
)

if not evidence_order_exact_630v18m:
    raise RuntimeError(
        "Dropout scenario-evidence order changed."
    )


mc_hash_checks_630v18m = {
    "count_63":
        len(mc_hashes_630v18m) == 63,

    "format_exact":
        bool(
            all(
                len(value) == 64
                and
                all(
                    char in "0123456789abcdef"
                    for char in value.lower()
                )
                for value in mc_hashes_630v18m
            )
        ),
}

if not all(mc_hash_checks_630v18m.values()):
    raise RuntimeError(
        "Dropout full-MC hash evidence changed."
    )


# =============================================================================
# 5) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18m = dict(
    FD004_DROPOUT_BLOCK_CONTRACT_630V18L
)

contract_recomputed_sha_630v18m = hashlib.sha256(
    json.dumps(
        contract_630v18m,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

contract_checks_630v18m = {
    "sha_exact":
        contract_recomputed_sha_630v18m
        ==
        resident_hashes_630v18m["block_contract"],

    "family":
        str(contract_630v18m.get("family"))
        == "dropout",

    "scenario_count":
        int(contract_630v18m.get("scenario_count"))
        == 63,

    "prediction_hash":
        str(
            contract_630v18m.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m["prediction"],

    "mc_std_hash":
        str(
            contract_630v18m.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m["mc_std"],

    "risk_hash":
        str(
            contract_630v18m.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m["final_risk"],

    "drift_closure_link":
        str(
            contract_630v18m.get(
                "drift_final_closure_sha256"
            )
        )
        ==
        str(
            FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K
        ),

    "full_mc_not_retained":
        bool(
            contract_630v18m.get(
                "full_mc_matrices_retained"
            )
        )
        is False,

    "base_unchanged":
        bool(
            contract_630v18m.get(
                "corruption_base_unchanged"
            )
        )
        is True,

    "model_state_unchanged":
        bool(
            contract_630v18m.get(
                "model_state_unchanged"
            )
        )
        is True,

    "model_mode_unchanged":
        bool(
            contract_630v18m.get(
                "model_mode_unchanged"
            )
        )
        is True,

    "dev_outcomes_unused":
        bool(
            contract_630v18m.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18m.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            contract_630v18m.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18m.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18m.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(contract_checks_630v18m.values()):
    failed_630v18m = [
        key
        for key, passed
        in contract_checks_630v18m.items()
        if not passed
    ]

    raise RuntimeError(
        "Dropout block-contract closure failed: "
        + ", ".join(failed_630v18m)
    )


# =============================================================================
# 6) CHECKPOINT BYTE INTEGRITY
# =============================================================================

prior_roundtrip_checks_630v18m = dict(
    FD004_DROPOUT_CHECKPOINT_ROUNDTRIP_CHECKS_630V18L
)

prior_roundtrip_all_true_630v18m = bool(
    prior_roundtrip_checks_630v18m
    and
    all(
        bool(value)
        for value in prior_roundtrip_checks_630v18m.values()
    )
)

if not prior_roundtrip_all_true_630v18m:
    raise RuntimeError(
        "Prior dropout checkpoint round-trip evidence changed."
    )


npz_path_630v18m = Path(
    FD004_DROPOUT_CHECKPOINT_NPZ_PATH_630V18L
)

json_path_630v18m = Path(
    FD004_DROPOUT_CHECKPOINT_JSON_PATH_630V18L
)

if not npz_path_630v18m.is_file():
    raise RuntimeError(
        "Dropout NPZ checkpoint file is missing."
    )

if not json_path_630v18m.is_file():
    raise RuntimeError(
        "Dropout JSON checkpoint file is missing."
    )


def _file_sha256_630v18m(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


checkpoint_byte_checks_630v18m = {
    "npz":
        _file_sha256_630v18m(
            npz_path_630v18m
        )
        ==
        resident_hashes_630v18m[
            "checkpoint_npz"
        ],

    "json":
        _file_sha256_630v18m(
            json_path_630v18m
        )
        ==
        resident_hashes_630v18m[
            "checkpoint_json"
        ],
}

if not all(checkpoint_byte_checks_630v18m.values()):
    raise RuntimeError(
        "Dropout checkpoint byte hash changed."
    )


# =============================================================================
# 7) SAFE NPZ REOPEN — allow_pickle=False
# =============================================================================

expected_npz_keys_630v18m = tuple(
    sorted(
        (
            "scenario_number",
            "scenario_id",
            "deterministic_prediction",
            "mc_std",
            "final_selective_risk",
            "full_mc_matrix_sha256",
        )
    )
)

with np.load(
    npz_path_630v18m,
    allow_pickle=False,
) as npz_630v18m:

    keys_630v18m = tuple(
        sorted(npz_630v18m.files)
    )

    if keys_630v18m != expected_npz_keys_630v18m:
        raise RuntimeError(
            "Dropout NPZ key set changed."
        )

    loaded_numbers_630v18m = np.asarray(
        npz_630v18m[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    loaded_id_raw_630v18m = np.asarray(
        npz_630v18m[
            "scenario_id"
        ]
    )

    loaded_ids_630v18m = (
        loaded_id_raw_630v18m
        .astype(str)
        .reshape(-1)
    )

    loaded_pred_630v18m = np.asarray(
        npz_630v18m[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    loaded_std_630v18m = np.asarray(
        npz_630v18m[
            "mc_std"
        ],
        dtype=np.float64,
    )

    loaded_risk_630v18m = np.asarray(
        npz_630v18m[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    loaded_mc_hashes_630v18m = tuple(
        str(x)
        for x in np.asarray(
            npz_630v18m[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


npz_checks_630v18m = {
    "scenario_id_non_object":
        bool(
            not loaded_id_raw_630v18m.dtype.hasobject
        ),

    "scenario_number":
        bool(
            np.array_equal(
                loaded_numbers_630v18m,
                manifest_630v18m[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                loaded_ids_630v18m,
                manifest_630v18m[
                    "scenario_id"
                ].astype(str).to_numpy(),
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                loaded_pred_630v18m,
                pred_630v18m,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                loaded_std_630v18m,
                std_630v18m,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                loaded_risk_630v18m,
                risk_630v18m,
            )
        ),

    "full_mc_hashes":
        loaded_mc_hashes_630v18m
        ==
        mc_hashes_630v18m,
}

if not all(npz_checks_630v18m.values()):
    failed_630v18m = [
        key
        for key, passed
        in npz_checks_630v18m.items()
        if not passed
    ]

    raise RuntimeError(
        "Dropout NPZ safe round-trip failed: "
        + ", ".join(failed_630v18m)
    )


# =============================================================================
# 8) JSON CHECKPOINT REVERIFY
# =============================================================================

with open(
    json_path_630v18m,
    "r",
    encoding="utf-8",
) as file:
    json_loaded_630v18m = json.load(file)


json_checks_630v18m = {
    "dataset":
        str(
            json_loaded_630v18m.get(
                "dataset"
            )
        )
        == "FD004",

    "stage":
        str(
            json_loaded_630v18m.get(
                "stage"
            )
        )
        == "6.30V-18L",

    "family":
        str(
            json_loaded_630v18m.get(
                "family"
            )
        )
        == "dropout",

    "scenario_count":
        int(
            json_loaded_630v18m.get(
                "scenario_count"
            )
        )
        == 63,

    "npz_sha":
        str(
            json_loaded_630v18m.get(
                "npz_sha256"
            )
        )
        ==
        resident_hashes_630v18m[
            "checkpoint_npz"
        ],

    "block_contract_sha":
        str(
            json_loaded_630v18m.get(
                "block_contract_sha256"
            )
        )
        ==
        resident_hashes_630v18m[
            "block_contract"
        ],

    "prediction_sha":
        str(
            json_loaded_630v18m.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m[
            "prediction"
        ],

    "mc_std_sha":
        str(
            json_loaded_630v18m.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m[
            "mc_std"
        ],

    "risk_sha":
        str(
            json_loaded_630v18m.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_hashes_630v18m[
            "final_risk"
        ],

    "scenario_evidence_count":
        len(
            json_loaded_630v18m.get(
                "scenario_evidence",
                [],
            )
        )
        == 63,

    "dev_outcomes_unused":
        bool(
            json_loaded_630v18m.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            json_loaded_630v18m.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            json_loaded_630v18m.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            json_loaded_630v18m.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            json_loaded_630v18m.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(json_checks_630v18m.values()):
    failed_630v18m = [
        key
        for key, passed
        in json_checks_630v18m.items()
        if not passed
    ]

    raise RuntimeError(
        "Dropout JSON verification failed: "
        + ", ".join(failed_630v18m)
    )


# =============================================================================
# 9) FINAL DROPOUT CLOSURE
# =============================================================================

FD004_DROPOUT_FINAL_CLOSURE_630V18M = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18M",

    "family":
        "dropout",

    "status":
        "DROPOUT_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        63,

    "prediction_block_sha256":
        resident_hashes_630v18m[
            "prediction"
        ],

    "mc_std_block_sha256":
        resident_hashes_630v18m[
            "mc_std"
        ],

    "final_risk_block_sha256":
        resident_hashes_630v18m[
            "final_risk"
        ],

    "dropout_block_contract_sha256":
        resident_hashes_630v18m[
            "block_contract"
        ],

    "safe_checkpoint_npz_sha256":
        resident_hashes_630v18m[
            "checkpoint_npz"
        ],

    "safe_checkpoint_json_sha256":
        resident_hashes_630v18m[
            "checkpoint_json"
        ],

    "safe_checkpoint_roundtrip_exact":
        True,

    "full_mc_matrix_hash_count":
        len(mc_hashes_630v18m),

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M = (
    hashlib.sha256(
        json.dumps(
            FD004_DROPOUT_FINAL_CLOSURE_630V18M,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DROPOUT_FAMILY_CLOSED_630V18M = True

FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18M = False
FD004_DEV_TARGETS_ERRORS_USED_630V18M = False
FD004_THRESHOLD_SELECTED_630V18M = False
FD004_POLICY_SELECTED_630V18M = False
FD004_TEST_CONTENTS_READ_630V18M = False
FD004_RUL_CONTENTS_READ_630V18M = False

all_checks_passed_630v18m = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18M — FD004 DROPOUT FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Dropout scenarios rerun           :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- RESIDENT SCIENTIFIC EVIDENCE ---")

print(
    "Prediction rehash exact           :",
    rehash_checks_630v18m[
        "prediction"
    ],
)

print(
    "MC STD rehash exact               :",
    rehash_checks_630v18m[
        "mc_std"
    ],
)

print(
    "Final-risk rehash exact           :",
    rehash_checks_630v18m[
        "final_risk"
    ],
)

print(
    "Manifest checks all true          :",
    all(
        manifest_checks_630v18m.values()
    ),
)

print(
    "Scenario evidence order exact     :",
    evidence_order_exact_630v18m,
)

print(
    "Full-MC hash evidence exact       :",
    all(
        mc_hash_checks_630v18m.values()
    ),
)


print("\n--- BLOCK CONTRACT ---")

for key, value in contract_checks_630v18m.items():
    print(
        f"{key:28s}:",
        value,
    )


print("\n--- SAFE CHECKPOINT ---")

print(
    "NPZ byte SHA exact                :",
    checkpoint_byte_checks_630v18m[
        "npz"
    ],
)

print(
    "JSON byte SHA exact               :",
    checkpoint_byte_checks_630v18m[
        "json"
    ],
)

print(
    "NPZ allow_pickle=False checks     :",
    npz_checks_630v18m,
)

print(
    "JSON checks                       :",
    json_checks_630v18m,
)

print(
    "Prior 18L round-trip all true     :",
    prior_roundtrip_all_true_630v18m,
)

print(
    "Drift closure link exact          :",
    drift_closure_link_exact_630v18m,
)


print("\n--- FINAL CLOSURE ---")

print(
    "Dropout family closed             :",
    FD004_DROPOUT_FAMILY_CLOSED_630V18M,
)

print(
    "Dropout final closure SHA-256     :",
    FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M,
)


print(
    "\nPASS — FD004 dropout is evidence-locked using only resident "
    "authoritative hashes and direct recomputation. No dropout "
    "scenario was rerun."
)

print(
    "Next: execute only the final 63 admissible stuck scenarios."
)

In [ ]:
# STEP 6.30V-18N — FD004 STUCK FAMILY EXECUTION
# 63 admissible stuck scenarios only.
#
# LABEL-FREE:
# corruption -> normalization -> windows -> deterministic prediction
# -> 50-pass MC dropout -> reliability chain
#
# NO DEV targets/errors.
# NO threshold/policy selection.
# NO TEST / official RUL access.
#
# Retain:
# - deterministic prediction
# - MC STD
# - final selective risk
# - SHA-256 of each full MC matrix, then discard full MC matrix.
#
# Checkpoint:
# - fixed-width Unicode scenario IDs
# - allow_pickle=False round-trip verification
# - persistence failure does NOT invalidate completed scientific execution.

import numpy as np
import pandas as pd
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18n = (
    "all_checks_passed_630v18m",
    "FD004_DROPOUT_FAMILY_CLOSED_630V18M",
    "FD004_DROPOUT_FINAL_CLOSURE_630V18M",
    "FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M",
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_DEV_CORRUPTION_BASE_630V17A",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_APPLY_SCENARIO_630V17A",
    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "FD004_DETERMINISTIC_PREDICTOR_630V18CR1",
    "fd004_mc_dropout_predict_630v4r2",
    "apply_fd004_final_reliability_chain_630v4r2",
    "MODEL_FD004_630I",
    "FD004_DEVICE_630I",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
    "FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1",
    "FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0",
)

missing_630v18n = [
    name
    for name in required_630v18n
    if name not in globals()
]

if missing_630v18n:
    raise RuntimeError(
        "Missing STEP 6.30V-18N authority object(s): "
        + ", ".join(missing_630v18n)
    )

if not bool(all_checks_passed_630v18m):
    raise RuntimeError(
        "STEP 6.30V-18M must remain passed."
    )

if not bool(FD004_DROPOUT_FAMILY_CLOSED_630V18M):
    raise RuntimeError(
        "Dropout family must remain evidence-locked before stuck."
    )

dropout_closure_recomputed_sha_630v18n = hashlib.sha256(
    json.dumps(
        FD004_DROPOUT_FINAL_CLOSURE_630V18M,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    dropout_closure_recomputed_sha_630v18n
    !=
    str(FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M)
):
    raise RuntimeError(
        "Resident dropout final-closure evidence changed."
    )


# =============================================================================
# 1) FREEZE STUCK BLOCK
# =============================================================================

execution_manifest_630v18n = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
)

stuck_manifest_630v18n = (
    execution_manifest_630v18n.loc[
        execution_manifest_630v18n[
            "family"
        ].astype(str)
        == "stuck"
    ]
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(stuck_manifest_630v18n) != 63:
    raise RuntimeError(
        "Expected exactly 63 admissible stuck scenarios."
    )

stuck_numbers_630v18n = (
    stuck_manifest_630v18n[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

if (
    int(stuck_numbers_630v18n.min()) != 379
    or
    int(stuck_numbers_630v18n.max()) != 441
):
    raise RuntimeError(
        "Frozen stuck scenario-number range changed."
    )

if not np.all(
    stuck_manifest_630v18n[
        "family"
    ].astype(str).to_numpy()
    ==
    "stuck"
):
    raise RuntimeError(
        "Non-stuck row entered the stuck block."
    )

if not stuck_manifest_630v18n[
    "requested_fraction"
].notna().all():
    raise RuntimeError(
        "Stuck manifest has missing requested_fraction values."
    )

requested_fractions_630v18n = (
    stuck_manifest_630v18n[
        "requested_fraction"
    ].to_numpy(dtype=np.float64)
)

if not np.all(
    (requested_fractions_630v18n > 0.0)
    &
    (requested_fractions_630v18n < 1.0)
):
    raise RuntimeError(
        "Stuck manifest has invalid requested_fraction values."
    )


# =============================================================================
# 2) FROZEN AUTHORITIES
# =============================================================================

base_630v18n = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

base_snapshot_630v18n = (
    base_630v18n.copy(deep=True)
)

if "regime_id" not in base_630v18n.columns:
    raise RuntimeError(
        "Frozen corruption base has no regime_id column."
    )

regime_ids_630v18n = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

if not np.array_equal(
    base_630v18n[
        "regime_id"
    ].to_numpy(dtype=np.int64),
    regime_ids_630v18n,
):
    raise RuntimeError(
        "Frozen corruption-base regime_id vector changed."
    )

expected_window_units_630v18n = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

expected_endpoint_cycles_630v18n = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)

if expected_window_units_630v18n.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV window-unit authority shape changed."
    )

if expected_endpoint_cycles_630v18n.shape != (10824,):
    raise RuntimeError(
        "Frozen DEV endpoint-cycle authority shape changed."
    )

model_630v18n = MODEL_FD004_630I
device_630v18n = FD004_DEVICE_630I

if bool(model_630v18n.training):
    raise RuntimeError(
        "FD004 model must be in eval mode before stuck execution."
    )


def _model_state_sha_630v18n(model):
    h = hashlib.sha256()
    state = model.state_dict()

    for key in sorted(state.keys()):
        tensor = (
            state[key]
            .detach()
            .cpu()
            .contiguous()
        )

        arr = tensor.numpy()

        h.update(key.encode("utf-8"))
        h.update(str(arr.dtype).encode("utf-8"))
        h.update(str(tuple(arr.shape)).encode("utf-8"))
        h.update(arr.tobytes(order="C"))

    return h.hexdigest()


model_state_before_630v18n = (
    _model_state_sha_630v18n(model_630v18n)
)

model_mode_before_630v18n = bool(
    model_630v18n.training
)


# =============================================================================
# 3) PREALLOCATE RETAINED PRODUCTS
# =============================================================================

n_scenarios_630v18n = 63
n_windows_630v18n = 10824

stuck_pred_matrix_630v18n = np.empty(
    (n_scenarios_630v18n, n_windows_630v18n),
    dtype=np.float32,
)

stuck_mc_std_matrix_630v18n = np.empty(
    (n_scenarios_630v18n, n_windows_630v18n),
    dtype=np.float64,
)

stuck_final_risk_matrix_630v18n = np.empty(
    (n_scenarios_630v18n, n_windows_630v18n),
    dtype=np.float64,
)

stuck_mc_matrix_hashes_630v18n = []
stuck_scenario_evidence_630v18n = []


# =============================================================================
# 4) EXECUTE 63 STUCK SCENARIOS
# =============================================================================

print("=" * 176)
print(
    "STEP 6.30V-18N — FD004 STUCK FAMILY EXECUTION "
    "(63 ADMISSIBLE SCENARIOS)"
)
print("=" * 176)

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nStarting frozen stuck block        :",
    n_scenarios_630v18n,
    "scenarios",
)


for row_index_630v18n, row_630v18n in (
    stuck_manifest_630v18n.iterrows()
):

    scenario_id_630v18n = str(
        row_630v18n["scenario_id"]
    )

    scenario_number_630v18n = int(
        row_630v18n["scenario_number"]
    )

    sensor_630v18n = str(
        row_630v18n["sensor"]
    )

    requested_fraction_630v18n = float(
        row_630v18n["requested_fraction"]
    )

    corrupted_raw_630v18n = (
        FD004_APPLY_SCENARIO_630V17A(
            base_630v18n,
            row_630v18n,
        )
    )

    if not isinstance(
        corrupted_raw_630v18n,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: dispatcher did not return DataFrame."
        )

    if corrupted_raw_630v18n.shape != (12274, 27):
        raise RuntimeError(
            f"{scenario_id_630v18n}: corrupted raw shape changed."
        )

    if not np.array_equal(
        corrupted_raw_630v18n[
            "regime_id"
        ].to_numpy(dtype=np.int64),
        regime_ids_630v18n,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: regime_id changed under stuck corruption."
        )

    if not np.isfinite(
        corrupted_raw_630v18n[
            sensor_630v18n
        ].to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18n}: stuck target sensor contains "
            "non-finite values."
        )

    normalized_630v18n = (
        normalize_fd004_raw_630v2(
            corrupted_raw_630v18n,
            regime_ids_630v18n,
        )
    )

    (
        X_630v18n,
        unit_ids_630v18n,
        endpoint_cycles_630v18n,
    ) = build_windows_fd004_630v2(
        normalized_630v18n
    )

    X_630v18n = np.asarray(
        X_630v18n,
        dtype=np.float32,
    )

    if X_630v18n.shape != (10824, 30, 24):
        raise RuntimeError(
            f"{scenario_id_630v18n}: window tensor shape changed."
        )

    if not np.isfinite(X_630v18n).all():
        raise RuntimeError(
            f"{scenario_id_630v18n}: non-finite model input."
        )

    if not np.array_equal(
        np.asarray(
            unit_ids_630v18n,
            dtype=np.int64,
        ).reshape(-1),
        expected_window_units_630v18n,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: window-unit membership changed."
        )

    if not np.array_equal(
        np.asarray(
            endpoint_cycles_630v18n,
            dtype=np.int64,
        ).reshape(-1),
        expected_endpoint_cycles_630v18n,
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: endpoint-cycle alignment changed."
        )

    deterministic_pred_630v18n = np.asarray(
        FD004_DETERMINISTIC_PREDICTOR_630V18CR1(
            X_630v18n,
            model_630v18n,
            device_630v18n,
            batch_size=512,
        ),
        dtype=np.float32,
    ).reshape(-1)

    if deterministic_pred_630v18n.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18n}: deterministic prediction shape changed."
        )

    if not np.isfinite(
        deterministic_pred_630v18n
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18n}: non-finite deterministic prediction."
        )

    (
        mc_matrix_630v18n,
        mc_mean_630v18n,
        mc_std_630v18n,
    ) = fd004_mc_dropout_predict_630v4r2(
        X_630v18n,
        model_630v18n,
        device_630v18n,
    )

    mc_matrix_630v18n = np.asarray(
        mc_matrix_630v18n,
        dtype=np.float32,
    )

    mc_std_630v18n = np.asarray(
        mc_std_630v18n,
        dtype=np.float64,
    ).reshape(-1)

    if mc_matrix_630v18n.shape != (50, 10824):
        raise RuntimeError(
            f"{scenario_id_630v18n}: MC matrix shape changed."
        )

    if mc_std_630v18n.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18n}: MC STD shape changed."
        )

    if not (
        np.isfinite(mc_matrix_630v18n).all()
        and
        np.isfinite(mc_std_630v18n).all()
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: non-finite MC output."
        )

    if np.any(mc_std_630v18n < 0.0):
        raise RuntimeError(
            f"{scenario_id_630v18n}: negative MC STD."
        )

    mc_matrix_sha_630v18n = str(
        sha256_array_630v2(
            mc_matrix_630v18n
        )
    )

    (
        phase_ids_630v18n,
        adjusted_630v18n,
        g1_630v18n,
        final_risk_630v18n,
    ) = apply_fd004_final_reliability_chain_630v4r2(
        deterministic_pred_630v18n,
        mc_std_630v18n,
    )

    final_risk_630v18n = np.asarray(
        final_risk_630v18n,
        dtype=np.float64,
    ).reshape(-1)

    if final_risk_630v18n.shape != (10824,):
        raise RuntimeError(
            f"{scenario_id_630v18n}: final risk shape changed."
        )

    if not np.isfinite(
        final_risk_630v18n
    ).all():
        raise RuntimeError(
            f"{scenario_id_630v18n}: non-finite final risk."
        )

    if np.any(
        (final_risk_630v18n < 0.0)
        |
        (final_risk_630v18n > 1.0)
    ):
        raise RuntimeError(
            f"{scenario_id_630v18n}: final risk escaped [0,1]."
        )

    stuck_pred_matrix_630v18n[
        row_index_630v18n
    ] = deterministic_pred_630v18n

    stuck_mc_std_matrix_630v18n[
        row_index_630v18n
    ] = mc_std_630v18n

    stuck_final_risk_matrix_630v18n[
        row_index_630v18n
    ] = final_risk_630v18n

    stuck_mc_matrix_hashes_630v18n.append(
        mc_matrix_sha_630v18n
    )

    stuck_scenario_evidence_630v18n.append(
        {
            "scenario_id":
                scenario_id_630v18n,

            "scenario_number":
                scenario_number_630v18n,

            "family":
                "stuck",

            "sensor":
                sensor_630v18n,

            "requested_fraction":
                requested_fraction_630v18n,

            "deterministic_prediction_sha256":
                str(
                    sha256_array_630v2(
                        deterministic_pred_630v18n
                    )
                ),

            "mc_matrix_sha256":
                mc_matrix_sha_630v18n,

            "mc_std_sha256":
                str(
                    sha256_array_630v2(
                        mc_std_630v18n
                    )
                ),

            "final_selective_risk_sha256":
                str(
                    sha256_array_630v2(
                        final_risk_630v18n
                    )
                ),
        }
    )

    del mc_matrix_630v18n
    del mc_mean_630v18n
    del phase_ids_630v18n
    del adjusted_630v18n
    del g1_630v18n
    del corrupted_raw_630v18n
    del normalized_630v18n
    del X_630v18n

    if (
        (row_index_630v18n + 1) == 1
        or
        (row_index_630v18n + 1) % 10 == 0
        or
        (row_index_630v18n + 1) == n_scenarios_630v18n
    ):
        print(
            f"Completed stuck scenario "
            f"{row_index_630v18n + 1:2d}/"
            f"{n_scenarios_630v18n}: "
            f"{scenario_id_630v18n}"
        )


# =============================================================================
# 5) POST-BLOCK SAFETY
# =============================================================================

if not np.isfinite(
    stuck_pred_matrix_630v18n
).all():
    raise RuntimeError(
        "Retained stuck deterministic predictions contain non-finite values."
    )

if not np.isfinite(
    stuck_mc_std_matrix_630v18n
).all():
    raise RuntimeError(
        "Retained stuck MC STD contains non-finite values."
    )

if not np.isfinite(
    stuck_final_risk_matrix_630v18n
).all():
    raise RuntimeError(
        "Retained stuck final risk contains non-finite values."
    )

if np.any(
    stuck_mc_std_matrix_630v18n < 0.0
):
    raise RuntimeError(
        "Retained stuck MC STD contains negative values."
    )

if np.any(
    (stuck_final_risk_matrix_630v18n < 0.0)
    |
    (stuck_final_risk_matrix_630v18n > 1.0)
):
    raise RuntimeError(
        "Retained stuck risk escaped [0,1]."
    )

if len(
    stuck_mc_matrix_hashes_630v18n
) != 63:
    raise RuntimeError(
        "Stuck MC matrix hash count changed."
    )

if len(
    stuck_scenario_evidence_630v18n
) != 63:
    raise RuntimeError(
        "Stuck scenario evidence count changed."
    )

base_unchanged_630v18n = bool(
    base_630v18n.equals(
        base_snapshot_630v18n
    )
)

if not base_unchanged_630v18n:
    raise RuntimeError(
        "Frozen corruption base changed during stuck execution."
    )

model_state_after_630v18n = (
    _model_state_sha_630v18n(
        model_630v18n
    )
)

model_mode_after_630v18n = bool(
    model_630v18n.training
)

model_state_unchanged_630v18n = bool(
    model_state_before_630v18n
    ==
    model_state_after_630v18n
)

model_mode_unchanged_630v18n = bool(
    model_mode_before_630v18n
    ==
    model_mode_after_630v18n
)

if not model_state_unchanged_630v18n:
    raise RuntimeError(
        "Model state changed during stuck block."
    )

if not model_mode_unchanged_630v18n:
    raise RuntimeError(
        "Model mode changed during stuck block."
    )


# =============================================================================
# 6) FREEZE STUCK EVIDENCE
# =============================================================================

FD004_STUCK_SCENARIO_MANIFEST_630V18N = (
    stuck_manifest_630v18n.copy()
)

FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N = (
    stuck_pred_matrix_630v18n
)

FD004_STUCK_MC_STD_630V18N = (
    stuck_mc_std_matrix_630v18n
)

FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N = (
    stuck_final_risk_matrix_630v18n
)

FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N = tuple(
    stuck_mc_matrix_hashes_630v18n
)

FD004_STUCK_SCENARIO_EVIDENCE_630V18N = tuple(
    stuck_scenario_evidence_630v18n
)

FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N = str(
    sha256_array_630v2(
        FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N
    )
)

FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N = str(
    sha256_array_630v2(
        FD004_STUCK_MC_STD_630V18N
    )
)

FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N = str(
    sha256_array_630v2(
        FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N
    )
)


stuck_block_contract_630v18n = {
    "dataset": "FD004",
    "stage": "6.30V-18N",
    "family": "stuck",
    "status": "STUCK_BLOCK_COMPLETE_LABEL_FREE",
    "scenario_count": 63,

    "scenario_number_min":
        int(stuck_numbers_630v18n.min()),

    "scenario_number_max":
        int(stuck_numbers_630v18n.max()),

    "deterministic_prediction_shape":
        tuple(
            FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N.shape
        ),

    "mc_std_shape":
        tuple(
            FD004_STUCK_MC_STD_630V18N.shape
        ),

    "final_selective_risk_shape":
        tuple(
            FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N.shape
        ),

    "full_mc_matrix_hash_count":
        len(
            FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N
        ),

    "full_mc_matrices_retained":
        False,

    "prediction_block_sha256":
        FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N,

    "mc_std_block_sha256":
        FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N,

    "final_risk_block_sha256":
        FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N,

    "dropout_final_closure_sha256":
        str(
            FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M
        ),

    "admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "deterministic_contract_sha256":
        str(
            FD004_DETERMINISTIC_INFERENCE_CONTRACT_SHA256_630V18CR1
        ),

    "helper_return_audit_sha256":
        str(
            FD004_EXECUTION_HELPER_RETURN_AUDIT_SHA256_630V18D0
        ),

    "corruption_base_unchanged":
        base_unchanged_630v18n,

    "model_state_unchanged":
        model_state_unchanged_630v18n,

    "model_mode_unchanged":
        model_mode_unchanged_630v18n,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_STUCK_BLOCK_CONTRACT_630V18N = (
    stuck_block_contract_630v18n
)

FD004_STUCK_BLOCK_CONTRACT_SHA256_630V18N = (
    hashlib.sha256(
        json.dumps(
            stuck_block_contract_630v18n,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 7) SAFE CHECKPOINT
# =============================================================================

checkpoint_saved_630v18n = False
checkpoint_roundtrip_exact_630v18n = False
checkpoint_error_630v18n = None

checkpoint_npz_path_630v18n = None
checkpoint_json_path_630v18n = None
checkpoint_npz_sha_630v18n = None
checkpoint_json_sha_630v18n = None
checkpoint_roundtrip_checks_630v18n = None

try:
    checkpoint_dir_630v18n = Path(
        r"C:\RUL_Reliability_Research\checkpoints"
    )

    checkpoint_dir_630v18n.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_npz_path_630v18n = (
        checkpoint_dir_630v18n
        /
        "fd004_stuck_block_630v18n_v1_safe.npz"
    )

    checkpoint_json_path_630v18n = (
        checkpoint_dir_630v18n
        /
        "fd004_stuck_block_630v18n_v1_safe.json"
    )

    npz_tmp_630v18n = (
        checkpoint_dir_630v18n
        /
        "fd004_stuck_block_630v18n_v1_safe.npz.tmp"
    )

    json_tmp_630v18n = (
        checkpoint_dir_630v18n
        /
        "fd004_stuck_block_630v18n_v1_safe.json.tmp"
    )

    scenario_id_list_630v18n = (
        stuck_manifest_630v18n[
            "scenario_id"
        ]
        .astype(str)
        .tolist()
    )

    scenario_id_max_len_630v18n = max(
        len(value)
        for value in scenario_id_list_630v18n
    )

    scenario_id_unicode_630v18n = np.asarray(
        scenario_id_list_630v18n,
        dtype=f"<U{scenario_id_max_len_630v18n}",
    )

    full_mc_hash_unicode_630v18n = np.asarray(
        FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N,
        dtype="<U64",
    )

    with open(
        npz_tmp_630v18n,
        "wb",
    ) as file:

        np.savez_compressed(
            file,

            scenario_number=
                stuck_manifest_630v18n[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),

            scenario_id=
                scenario_id_unicode_630v18n,

            deterministic_prediction=
                FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N,

            mc_std=
                FD004_STUCK_MC_STD_630V18N,

            final_selective_risk=
                FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N,

            full_mc_matrix_sha256=
                full_mc_hash_unicode_630v18n,
        )

    os.replace(
        npz_tmp_630v18n,
        checkpoint_npz_path_630v18n,
    )


    def _file_sha256_630v18n(path):
        h = hashlib.sha256()

        with open(
            path,
            "rb",
        ) as file:

            while True:
                chunk = file.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                h.update(chunk)

        return h.hexdigest()


    checkpoint_npz_sha_630v18n = (
        _file_sha256_630v18n(
            checkpoint_npz_path_630v18n
        )
    )


    checkpoint_metadata_630v18n = {
        "dataset": "FD004",
        "stage": "6.30V-18N",
        "family": "stuck",
        "status": "SAFE_SERIALIZATION_CHECKPOINT",
        "scenario_count": 63,

        "scenario_id_dtype":
            str(
                scenario_id_unicode_630v18n.dtype
            ),

        "block_contract_sha256":
            FD004_STUCK_BLOCK_CONTRACT_SHA256_630V18N,

        "prediction_block_sha256":
            FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N,

        "mc_std_block_sha256":
            FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N,

        "final_risk_block_sha256":
            FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N,

        "npz_sha256":
            checkpoint_npz_sha_630v18n,

        "scenario_evidence":
            list(
                FD004_STUCK_SCENARIO_EVIDENCE_630V18N
            ),

        "dev_targets_errors_used":
            False,

        "threshold_selected":
            False,

        "policy_selected":
            False,

        "test_contents_read":
            False,

        "official_rul_contents_read":
            False,
    }

    with open(
        json_tmp_630v18n,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            checkpoint_metadata_630v18n,
            file,
            sort_keys=True,
            indent=2,
            default=str,
        )

    os.replace(
        json_tmp_630v18n,
        checkpoint_json_path_630v18n,
    )

    checkpoint_json_sha_630v18n = (
        _file_sha256_630v18n(
            checkpoint_json_path_630v18n
        )
    )

    with np.load(
        checkpoint_npz_path_630v18n,
        allow_pickle=False,
    ) as checkpoint_npz_630v18n:

        loaded_numbers_630v18n = np.asarray(
            checkpoint_npz_630v18n[
                "scenario_number"
            ],
            dtype=np.int64,
        ).reshape(-1)

        loaded_id_raw_630v18n = np.asarray(
            checkpoint_npz_630v18n[
                "scenario_id"
            ]
        )

        loaded_ids_630v18n = (
            loaded_id_raw_630v18n
            .astype(str)
            .reshape(-1)
        )

        loaded_pred_630v18n = np.asarray(
            checkpoint_npz_630v18n[
                "deterministic_prediction"
            ],
            dtype=np.float32,
        )

        loaded_std_630v18n = np.asarray(
            checkpoint_npz_630v18n[
                "mc_std"
            ],
            dtype=np.float64,
        )

        loaded_risk_630v18n = np.asarray(
            checkpoint_npz_630v18n[
                "final_selective_risk"
            ],
            dtype=np.float64,
        )

        loaded_mc_hashes_630v18n = tuple(
            str(x)
            for x in np.asarray(
                checkpoint_npz_630v18n[
                    "full_mc_matrix_sha256"
                ]
            ).reshape(-1)
        )


    checkpoint_roundtrip_checks_630v18n = {
        "scenario_id_non_object":
            bool(
                not loaded_id_raw_630v18n.dtype.hasobject
            ),

        "scenario_number":
            bool(
                np.array_equal(
                    loaded_numbers_630v18n,
                    stuck_manifest_630v18n[
                        "scenario_number"
                    ].to_numpy(dtype=np.int64),
                )
            ),

        "scenario_id":
            bool(
                np.array_equal(
                    loaded_ids_630v18n,
                    scenario_id_unicode_630v18n,
                )
            ),

        "prediction":
            bool(
                np.array_equal(
                    loaded_pred_630v18n,
                    FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N,
                )
            ),

        "mc_std":
            bool(
                np.array_equal(
                    loaded_std_630v18n,
                    FD004_STUCK_MC_STD_630V18N,
                )
            ),

        "final_risk":
            bool(
                np.array_equal(
                    loaded_risk_630v18n,
                    FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N,
                )
            ),

        "full_mc_hashes":
            loaded_mc_hashes_630v18n
            ==
            tuple(
                FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N
            ),
    }

    checkpoint_roundtrip_exact_630v18n = bool(
        all(
            checkpoint_roundtrip_checks_630v18n.values()
        )
    )

    if not checkpoint_roundtrip_exact_630v18n:
        raise RuntimeError(
            "Stuck safe checkpoint round-trip verification failed."
        )

    checkpoint_saved_630v18n = True

except Exception as exc_630v18n:
    checkpoint_error_630v18n = repr(
        exc_630v18n
    )


# =============================================================================
# 8) FREEZE CHECKPOINT STATUS
# =============================================================================

FD004_STUCK_CHECKPOINT_SAVED_630V18N = bool(
    checkpoint_saved_630v18n
)

FD004_STUCK_CHECKPOINT_ROUNDTRIP_EXACT_630V18N = bool(
    checkpoint_roundtrip_exact_630v18n
)

FD004_STUCK_CHECKPOINT_NPZ_PATH_630V18N = (
    None
    if checkpoint_npz_path_630v18n is None
    else str(checkpoint_npz_path_630v18n)
)

FD004_STUCK_CHECKPOINT_JSON_PATH_630V18N = (
    None
    if checkpoint_json_path_630v18n is None
    else str(checkpoint_json_path_630v18n)
)

FD004_STUCK_CHECKPOINT_NPZ_SHA256_630V18N = (
    checkpoint_npz_sha_630v18n
)

FD004_STUCK_CHECKPOINT_JSON_SHA256_630V18N = (
    checkpoint_json_sha_630v18n
)

FD004_STUCK_CHECKPOINT_ROUNDTRIP_CHECKS_630V18N = (
    checkpoint_roundtrip_checks_630v18n
)

FD004_STUCK_CHECKPOINT_ERROR_630V18N = (
    checkpoint_error_630v18n
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD004_STUCK_BLOCK_COMPLETE_630V18N = True
FD004_STUCK_SCENARIOS_EXECUTED_630V18N = 63

FD004_DEV_TARGETS_ERRORS_USED_630V18N = False
FD004_THRESHOLD_SELECTED_630V18N = False
FD004_POLICY_SELECTED_630V18N = False
FD004_TEST_CONTENTS_READ_630V18N = False
FD004_RUL_CONTENTS_READ_630V18N = False

all_checks_passed_630v18n = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("\n--- STUCK BLOCK COMPLETE ---")

print(
    "Stuck scenarios executed          :",
    FD004_STUCK_SCENARIOS_EXECUTED_630V18N,
)

print(
    "Deterministic retained shape      :",
    FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N.shape,
)

print(
    "MC STD retained shape             :",
    FD004_STUCK_MC_STD_630V18N.shape,
)

print(
    "Final risk retained shape         :",
    FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N.shape,
)

print(
    "Full MC matrix hashes retained    :",
    len(
        FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N
    ),
)

print(
    "Full MC matrices retained         :",
    False,
)

print(
    "Corruption base unchanged         :",
    base_unchanged_630v18n,
)

print(
    "Model state unchanged             :",
    model_state_unchanged_630v18n,
)

print(
    "Model mode unchanged              :",
    model_mode_unchanged_630v18n,
)

print(
    "Prediction block SHA-256          :",
    FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N,
)

print(
    "MC STD block SHA-256              :",
    FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N,
)

print(
    "Final risk block SHA-256          :",
    FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N,
)

print(
    "Stuck block contract SHA-256      :",
    FD004_STUCK_BLOCK_CONTRACT_SHA256_630V18N,
)


print("\n--- LABEL-FREE BLOCK SUMMARY ---")

print(
    "MC STD min / median / mean / max  :",
    (
        float(np.min(FD004_STUCK_MC_STD_630V18N)),
        float(np.median(FD004_STUCK_MC_STD_630V18N)),
        float(np.mean(FD004_STUCK_MC_STD_630V18N)),
        float(np.max(FD004_STUCK_MC_STD_630V18N)),
    ),
)

print(
    "Risk min / median / mean / max    :",
    (
        float(np.min(FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N)),
        float(np.median(FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N)),
        float(np.mean(FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N)),
        float(np.max(FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N)),
    ),
)


print("\n--- SAFE CHECKPOINT ---")

print(
    "Checkpoint saved                  :",
    FD004_STUCK_CHECKPOINT_SAVED_630V18N,
)

print(
    "allow_pickle=False round-trip     :",
    FD004_STUCK_CHECKPOINT_ROUNDTRIP_EXACT_630V18N,
)

print(
    "NPZ path                          :",
    FD004_STUCK_CHECKPOINT_NPZ_PATH_630V18N,
)

print(
    "JSON path                         :",
    FD004_STUCK_CHECKPOINT_JSON_PATH_630V18N,
)

print(
    "NPZ SHA-256                       :",
    FD004_STUCK_CHECKPOINT_NPZ_SHA256_630V18N,
)

print(
    "JSON SHA-256                      :",
    FD004_STUCK_CHECKPOINT_JSON_SHA256_630V18N,
)

print(
    "Round-trip checks                 :",
    FD004_STUCK_CHECKPOINT_ROUNDTRIP_CHECKS_630V18N,
)

print(
    "Checkpoint error                  :",
    FD004_STUCK_CHECKPOINT_ERROR_630V18N,
)


print("\n--- LEAKAGE / POLICY GUARDS ---")

print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nPASS — the final 63 admissible FD004 stuck scenarios are "
    "complete under the frozen label-free execution contract."
)

if (
    checkpoint_saved_630v18n
    and
    checkpoint_roundtrip_exact_630v18n
):
    print(
        "The stuck block also has a verified pickle-free checkpoint. "
        "Next: close stuck evidence and then close the full "
        "381-scenario controlled-corruption execution."
    )
else:
    print(
        "The scientific stuck block is complete, but checkpoint "
        "persistence/verification failed. Do NOT rerun stuck; "
        "repair persistence only."
    )

In [ ]:
# STEP 6.30V-18O — FD004 STUCK FAMILY FINAL EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
# NO corruption / normalization / windows.
# NO deterministic inference / MC / reliability.
# NO DEV targets/errors.
# NO threshold/policy.
# NO TEST / official RUL.
# NO checkpoint rewrite.
#
# IMPORTANT:
# No manually transcribed stuck-result hashes are used.

import numpy as np
import hashlib
import json
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18o = (
    "all_checks_passed_630v18n",
    "FD004_STUCK_BLOCK_COMPLETE_630V18N",
    "FD004_STUCK_SCENARIOS_EXECUTED_630V18N",
    "FD004_STUCK_SCENARIO_MANIFEST_630V18N",
    "FD004_STUCK_SCENARIO_EVIDENCE_630V18N",
    "FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N",
    "FD004_STUCK_MC_STD_630V18N",
    "FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N",
    "FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N",
    "FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N",
    "FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N",
    "FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N",
    "FD004_STUCK_BLOCK_CONTRACT_630V18N",
    "FD004_STUCK_BLOCK_CONTRACT_SHA256_630V18N",
    "FD004_STUCK_CHECKPOINT_SAVED_630V18N",
    "FD004_STUCK_CHECKPOINT_ROUNDTRIP_EXACT_630V18N",
    "FD004_STUCK_CHECKPOINT_ROUNDTRIP_CHECKS_630V18N",
    "FD004_STUCK_CHECKPOINT_NPZ_PATH_630V18N",
    "FD004_STUCK_CHECKPOINT_JSON_PATH_630V18N",
    "FD004_STUCK_CHECKPOINT_NPZ_SHA256_630V18N",
    "FD004_STUCK_CHECKPOINT_JSON_SHA256_630V18N",
    "FD004_DROPOUT_FINAL_CLOSURE_630V18M",
    "FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M",
    "sha256_array_630v2",
)

missing_630v18o = [
    name
    for name in required_630v18o
    if name not in globals()
]

if missing_630v18o:
    raise RuntimeError(
        "Missing STEP 6.30V-18O authority object(s): "
        + ", ".join(missing_630v18o)
    )

if not bool(all_checks_passed_630v18n):
    raise RuntimeError(
        "STEP 6.30V-18N must remain passed."
    )

if not bool(FD004_STUCK_BLOCK_COMPLETE_630V18N):
    raise RuntimeError(
        "Stuck block is not marked complete."
    )

if int(FD004_STUCK_SCENARIOS_EXECUTED_630V18N) != 63:
    raise RuntimeError(
        "Stuck executed-scenario count changed."
    )

if not bool(FD004_STUCK_CHECKPOINT_SAVED_630V18N):
    raise RuntimeError(
        "Stuck checkpoint is not marked saved."
    )

if not bool(FD004_STUCK_CHECKPOINT_ROUNDTRIP_EXACT_630V18N):
    raise RuntimeError(
        "Stuck checkpoint is not marked round-trip exact."
    )


# =============================================================================
# 1) VERIFY PRIOR DROPOUT CLOSURE LINK
# =============================================================================

dropout_closure_recomputed_sha_630v18o = hashlib.sha256(
    json.dumps(
        FD004_DROPOUT_FINAL_CLOSURE_630V18M,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

dropout_closure_link_exact_630v18o = bool(
    dropout_closure_recomputed_sha_630v18o
    ==
    str(FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M)
)

if not dropout_closure_link_exact_630v18o:
    raise RuntimeError(
        "Resident dropout final-closure evidence changed."
    )


# =============================================================================
# 2) RESIDENT SCIENTIFIC EVIDENCE
# =============================================================================

manifest_630v18o = (
    FD004_STUCK_SCENARIO_MANIFEST_630V18N
    .copy()
    .reset_index(drop=True)
)

pred_630v18o = np.asarray(
    FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N,
    dtype=np.float32,
)

std_630v18o = np.asarray(
    FD004_STUCK_MC_STD_630V18N,
    dtype=np.float64,
)

risk_630v18o = np.asarray(
    FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N,
    dtype=np.float64,
)

mc_hashes_630v18o = tuple(
    str(x)
    for x in FD004_STUCK_FULL_MC_MATRIX_SHA256_630V18N
)

scenario_evidence_630v18o = tuple(
    FD004_STUCK_SCENARIO_EVIDENCE_630V18N
)


shape_checks_630v18o = {
    "prediction":
        pred_630v18o.shape == (63, 10824),

    "mc_std":
        std_630v18o.shape == (63, 10824),

    "final_risk":
        risk_630v18o.shape == (63, 10824),
}

validity_checks_630v18o = {
    "prediction_finite":
        bool(np.isfinite(pred_630v18o).all()),

    "mc_std_finite":
        bool(np.isfinite(std_630v18o).all()),

    "mc_std_nonnegative":
        bool(np.all(std_630v18o >= 0.0)),

    "risk_finite":
        bool(np.isfinite(risk_630v18o).all()),

    "risk_in_unit_interval":
        bool(
            np.all(
                (risk_630v18o >= 0.0)
                &
                (risk_630v18o <= 1.0)
            )
        ),
}

if not all(shape_checks_630v18o.values()):
    raise RuntimeError(
        "Stuck retained-array shape changed."
    )

if not all(validity_checks_630v18o.values()):
    raise RuntimeError(
        "Stuck retained-array validity check failed."
    )


# =============================================================================
# 3) RESIDENT AUTHORITATIVE HASHES — NO MANUAL VALUES
# =============================================================================

resident_hashes_630v18o = {
    "prediction":
        str(FD004_STUCK_BLOCK_PREDICTIONS_SHA256_630V18N),

    "mc_std":
        str(FD004_STUCK_BLOCK_MC_STD_SHA256_630V18N),

    "final_risk":
        str(FD004_STUCK_BLOCK_FINAL_RISK_SHA256_630V18N),

    "block_contract":
        str(FD004_STUCK_BLOCK_CONTRACT_SHA256_630V18N),

    "checkpoint_npz":
        str(FD004_STUCK_CHECKPOINT_NPZ_SHA256_630V18N),

    "checkpoint_json":
        str(FD004_STUCK_CHECKPOINT_JSON_SHA256_630V18N),
}

for key_630v18o, value_630v18o in resident_hashes_630v18o.items():
    if (
        len(value_630v18o) != 64
        or
        not all(
            char in "0123456789abcdef"
            for char in value_630v18o.lower()
        )
    ):
        raise RuntimeError(
            f"Resident {key_630v18o} authority is not a valid SHA-256."
        )


rehash_checks_630v18o = {
    "prediction":
        str(
            sha256_array_630v2(
                pred_630v18o
            )
        )
        ==
        resident_hashes_630v18o["prediction"],

    "mc_std":
        str(
            sha256_array_630v2(
                std_630v18o
            )
        )
        ==
        resident_hashes_630v18o["mc_std"],

    "final_risk":
        str(
            sha256_array_630v2(
                risk_630v18o
            )
        )
        ==
        resident_hashes_630v18o["final_risk"],
}

if not all(rehash_checks_630v18o.values()):
    failed_630v18o = [
        key
        for key, passed
        in rehash_checks_630v18o.items()
        if not passed
    ]

    raise RuntimeError(
        "Resident stuck scientific rehash failed: "
        + ", ".join(failed_630v18o)
    )


# =============================================================================
# 4) MANIFEST / SCENARIO EVIDENCE
# =============================================================================

manifest_checks_630v18o = {
    "row_count":
        len(manifest_630v18o) == 63,

    "family_stuck":
        bool(
            np.all(
                manifest_630v18o[
                    "family"
                ].astype(str).to_numpy()
                ==
                "stuck"
            )
        ),

    "number_min":
        int(
            manifest_630v18o[
                "scenario_number"
            ].min()
        )
        == 379,

    "number_max":
        int(
            manifest_630v18o[
                "scenario_number"
            ].max()
        )
        == 441,

    "numbers_strictly_increasing":
        bool(
            np.all(
                np.diff(
                    manifest_630v18o[
                        "scenario_number"
                    ].to_numpy(dtype=np.int64)
                )
                > 0
            )
        ),

    "requested_fraction_complete":
        bool(
            manifest_630v18o[
                "requested_fraction"
            ].notna().all()
        ),

    "requested_fraction_valid":
        bool(
            np.all(
                (
                    manifest_630v18o[
                        "requested_fraction"
                    ].to_numpy(dtype=np.float64)
                    > 0.0
                )
                &
                (
                    manifest_630v18o[
                        "requested_fraction"
                    ].to_numpy(dtype=np.float64)
                    < 1.0
                )
            )
        ),
}

if not all(manifest_checks_630v18o.values()):
    raise RuntimeError(
        "Stuck manifest closure check failed."
    )


if len(scenario_evidence_630v18o) != 63:
    raise RuntimeError(
        "Stuck scenario-evidence count changed."
    )

evidence_ids_630v18o = np.asarray(
    [
        str(row["scenario_id"])
        for row in scenario_evidence_630v18o
    ],
    dtype=object,
)

evidence_numbers_630v18o = np.asarray(
    [
        int(row["scenario_number"])
        for row in scenario_evidence_630v18o
    ],
    dtype=np.int64,
)

evidence_order_exact_630v18o = bool(
    np.array_equal(
        evidence_ids_630v18o,
        manifest_630v18o[
            "scenario_id"
        ].astype(str).to_numpy(),
    )
    and
    np.array_equal(
        evidence_numbers_630v18o,
        manifest_630v18o[
            "scenario_number"
        ].to_numpy(dtype=np.int64),
    )
)

if not evidence_order_exact_630v18o:
    raise RuntimeError(
        "Stuck scenario-evidence order changed."
    )


mc_hash_checks_630v18o = {
    "count_63":
        len(mc_hashes_630v18o) == 63,

    "format_exact":
        bool(
            all(
                len(value) == 64
                and
                all(
                    char in "0123456789abcdef"
                    for char in value.lower()
                )
                for value in mc_hashes_630v18o
            )
        ),
}

if not all(mc_hash_checks_630v18o.values()):
    raise RuntimeError(
        "Stuck full-MC hash evidence changed."
    )


# =============================================================================
# 5) BLOCK CONTRACT SELF-CONSISTENCY
# =============================================================================

contract_630v18o = dict(
    FD004_STUCK_BLOCK_CONTRACT_630V18N
)

contract_recomputed_sha_630v18o = hashlib.sha256(
    json.dumps(
        contract_630v18o,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

contract_checks_630v18o = {
    "sha_exact":
        contract_recomputed_sha_630v18o
        ==
        resident_hashes_630v18o["block_contract"],

    "family":
        str(contract_630v18o.get("family"))
        == "stuck",

    "scenario_count":
        int(contract_630v18o.get("scenario_count"))
        == 63,

    "prediction_hash":
        str(
            contract_630v18o.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o["prediction"],

    "mc_std_hash":
        str(
            contract_630v18o.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o["mc_std"],

    "risk_hash":
        str(
            contract_630v18o.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o["final_risk"],

    "dropout_closure_link":
        str(
            contract_630v18o.get(
                "dropout_final_closure_sha256"
            )
        )
        ==
        str(
            FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M
        ),

    "full_mc_not_retained":
        bool(
            contract_630v18o.get(
                "full_mc_matrices_retained"
            )
        )
        is False,

    "base_unchanged":
        bool(
            contract_630v18o.get(
                "corruption_base_unchanged"
            )
        )
        is True,

    "model_state_unchanged":
        bool(
            contract_630v18o.get(
                "model_state_unchanged"
            )
        )
        is True,

    "model_mode_unchanged":
        bool(
            contract_630v18o.get(
                "model_mode_unchanged"
            )
        )
        is True,

    "dev_outcomes_unused":
        bool(
            contract_630v18o.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            contract_630v18o.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            contract_630v18o.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            contract_630v18o.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            contract_630v18o.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(contract_checks_630v18o.values()):
    failed_630v18o = [
        key
        for key, passed
        in contract_checks_630v18o.items()
        if not passed
    ]

    raise RuntimeError(
        "Stuck block-contract closure failed: "
        + ", ".join(failed_630v18o)
    )


# =============================================================================
# 6) CHECKPOINT BYTE INTEGRITY
# =============================================================================

prior_roundtrip_checks_630v18o = dict(
    FD004_STUCK_CHECKPOINT_ROUNDTRIP_CHECKS_630V18N
)

prior_roundtrip_all_true_630v18o = bool(
    prior_roundtrip_checks_630v18o
    and
    all(
        bool(value)
        for value in prior_roundtrip_checks_630v18o.values()
    )
)

if not prior_roundtrip_all_true_630v18o:
    raise RuntimeError(
        "Prior stuck checkpoint round-trip evidence changed."
    )


npz_path_630v18o = Path(
    FD004_STUCK_CHECKPOINT_NPZ_PATH_630V18N
)

json_path_630v18o = Path(
    FD004_STUCK_CHECKPOINT_JSON_PATH_630V18N
)

if not npz_path_630v18o.is_file():
    raise RuntimeError(
        "Stuck NPZ checkpoint file is missing."
    )

if not json_path_630v18o.is_file():
    raise RuntimeError(
        "Stuck JSON checkpoint file is missing."
    )


def _file_sha256_630v18o(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


checkpoint_byte_checks_630v18o = {
    "npz":
        _file_sha256_630v18o(
            npz_path_630v18o
        )
        ==
        resident_hashes_630v18o[
            "checkpoint_npz"
        ],

    "json":
        _file_sha256_630v18o(
            json_path_630v18o
        )
        ==
        resident_hashes_630v18o[
            "checkpoint_json"
        ],
}

if not all(checkpoint_byte_checks_630v18o.values()):
    raise RuntimeError(
        "Stuck checkpoint byte hash changed."
    )


# =============================================================================
# 7) SAFE NPZ REOPEN — allow_pickle=False
# =============================================================================

expected_npz_keys_630v18o = tuple(
    sorted(
        (
            "scenario_number",
            "scenario_id",
            "deterministic_prediction",
            "mc_std",
            "final_selective_risk",
            "full_mc_matrix_sha256",
        )
    )
)

with np.load(
    npz_path_630v18o,
    allow_pickle=False,
) as npz_630v18o:

    keys_630v18o = tuple(
        sorted(npz_630v18o.files)
    )

    if keys_630v18o != expected_npz_keys_630v18o:
        raise RuntimeError(
            "Stuck NPZ key set changed."
        )

    loaded_numbers_630v18o = np.asarray(
        npz_630v18o[
            "scenario_number"
        ],
        dtype=np.int64,
    ).reshape(-1)

    loaded_id_raw_630v18o = np.asarray(
        npz_630v18o[
            "scenario_id"
        ]
    )

    loaded_ids_630v18o = (
        loaded_id_raw_630v18o
        .astype(str)
        .reshape(-1)
    )

    loaded_pred_630v18o = np.asarray(
        npz_630v18o[
            "deterministic_prediction"
        ],
        dtype=np.float32,
    )

    loaded_std_630v18o = np.asarray(
        npz_630v18o[
            "mc_std"
        ],
        dtype=np.float64,
    )

    loaded_risk_630v18o = np.asarray(
        npz_630v18o[
            "final_selective_risk"
        ],
        dtype=np.float64,
    )

    loaded_mc_hashes_630v18o = tuple(
        str(x)
        for x in np.asarray(
            npz_630v18o[
                "full_mc_matrix_sha256"
            ]
        ).reshape(-1)
    )


npz_checks_630v18o = {
    "scenario_id_non_object":
        bool(
            not loaded_id_raw_630v18o.dtype.hasobject
        ),

    "scenario_number":
        bool(
            np.array_equal(
                loaded_numbers_630v18o,
                manifest_630v18o[
                    "scenario_number"
                ].to_numpy(dtype=np.int64),
            )
        ),

    "scenario_id":
        bool(
            np.array_equal(
                loaded_ids_630v18o,
                manifest_630v18o[
                    "scenario_id"
                ].astype(str).to_numpy(),
            )
        ),

    "prediction":
        bool(
            np.array_equal(
                loaded_pred_630v18o,
                pred_630v18o,
            )
        ),

    "mc_std":
        bool(
            np.array_equal(
                loaded_std_630v18o,
                std_630v18o,
            )
        ),

    "final_risk":
        bool(
            np.array_equal(
                loaded_risk_630v18o,
                risk_630v18o,
            )
        ),

    "full_mc_hashes":
        loaded_mc_hashes_630v18o
        ==
        mc_hashes_630v18o,
}

if not all(npz_checks_630v18o.values()):
    failed_630v18o = [
        key
        for key, passed
        in npz_checks_630v18o.items()
        if not passed
    ]

    raise RuntimeError(
        "Stuck NPZ safe round-trip failed: "
        + ", ".join(failed_630v18o)
    )


# =============================================================================
# 8) JSON CHECKPOINT REVERIFY
# =============================================================================

with open(
    json_path_630v18o,
    "r",
    encoding="utf-8",
) as file:
    json_loaded_630v18o = json.load(file)


json_checks_630v18o = {
    "dataset":
        str(
            json_loaded_630v18o.get(
                "dataset"
            )
        )
        == "FD004",

    "stage":
        str(
            json_loaded_630v18o.get(
                "stage"
            )
        )
        == "6.30V-18N",

    "family":
        str(
            json_loaded_630v18o.get(
                "family"
            )
        )
        == "stuck",

    "scenario_count":
        int(
            json_loaded_630v18o.get(
                "scenario_count"
            )
        )
        == 63,

    "npz_sha":
        str(
            json_loaded_630v18o.get(
                "npz_sha256"
            )
        )
        ==
        resident_hashes_630v18o[
            "checkpoint_npz"
        ],

    "block_contract_sha":
        str(
            json_loaded_630v18o.get(
                "block_contract_sha256"
            )
        )
        ==
        resident_hashes_630v18o[
            "block_contract"
        ],

    "prediction_sha":
        str(
            json_loaded_630v18o.get(
                "prediction_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o[
            "prediction"
        ],

    "mc_std_sha":
        str(
            json_loaded_630v18o.get(
                "mc_std_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o[
            "mc_std"
        ],

    "risk_sha":
        str(
            json_loaded_630v18o.get(
                "final_risk_block_sha256"
            )
        )
        ==
        resident_hashes_630v18o[
            "final_risk"
        ],

    "scenario_evidence_count":
        len(
            json_loaded_630v18o.get(
                "scenario_evidence",
                [],
            )
        )
        == 63,

    "dev_outcomes_unused":
        bool(
            json_loaded_630v18o.get(
                "dev_targets_errors_used"
            )
        )
        is False,

    "threshold_unselected":
        bool(
            json_loaded_630v18o.get(
                "threshold_selected"
            )
        )
        is False,

    "policy_unselected":
        bool(
            json_loaded_630v18o.get(
                "policy_selected"
            )
        )
        is False,

    "test_unread":
        bool(
            json_loaded_630v18o.get(
                "test_contents_read"
            )
        )
        is False,

    "rul_unread":
        bool(
            json_loaded_630v18o.get(
                "official_rul_contents_read"
            )
        )
        is False,
}

if not all(json_checks_630v18o.values()):
    failed_630v18o = [
        key
        for key, passed
        in json_checks_630v18o.items()
        if not passed
    ]

    raise RuntimeError(
        "Stuck JSON verification failed: "
        + ", ".join(failed_630v18o)
    )


# =============================================================================
# 9) FINAL STUCK CLOSURE
# =============================================================================

FD004_STUCK_FINAL_CLOSURE_630V18O = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18O",

    "family":
        "stuck",

    "status":
        "STUCK_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "scientific_scenario_count":
        63,

    "prediction_block_sha256":
        resident_hashes_630v18o[
            "prediction"
        ],

    "mc_std_block_sha256":
        resident_hashes_630v18o[
            "mc_std"
        ],

    "final_risk_block_sha256":
        resident_hashes_630v18o[
            "final_risk"
        ],

    "stuck_block_contract_sha256":
        resident_hashes_630v18o[
            "block_contract"
        ],

    "safe_checkpoint_npz_sha256":
        resident_hashes_630v18o[
            "checkpoint_npz"
        ],

    "safe_checkpoint_json_sha256":
        resident_hashes_630v18o[
            "checkpoint_json"
        ],

    "safe_checkpoint_roundtrip_exact":
        True,

    "full_mc_matrix_hash_count":
        len(mc_hashes_630v18o),

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_STUCK_FINAL_CLOSURE_SHA256_630V18O = (
    hashlib.sha256(
        json.dumps(
            FD004_STUCK_FINAL_CLOSURE_630V18O,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_STUCK_FAMILY_CLOSED_630V18O = True

FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18O = False
FD004_DEV_TARGETS_ERRORS_USED_630V18O = False
FD004_THRESHOLD_SELECTED_630V18O = False
FD004_POLICY_SELECTED_630V18O = False
FD004_TEST_CONTENTS_READ_630V18O = False
FD004_RUL_CONTENTS_READ_630V18O = False

all_checks_passed_630v18o = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18O — FD004 STUCK FAMILY "
    "FINAL EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Stuck scenarios rerun             :", 0)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- RESIDENT SCIENTIFIC EVIDENCE ---")

print(
    "Prediction rehash exact           :",
    rehash_checks_630v18o[
        "prediction"
    ],
)

print(
    "MC STD rehash exact               :",
    rehash_checks_630v18o[
        "mc_std"
    ],
)

print(
    "Final-risk rehash exact           :",
    rehash_checks_630v18o[
        "final_risk"
    ],
)

print(
    "Manifest checks all true          :",
    all(
        manifest_checks_630v18o.values()
    ),
)

print(
    "Scenario evidence order exact     :",
    evidence_order_exact_630v18o,
)

print(
    "Full-MC hash evidence exact       :",
    all(
        mc_hash_checks_630v18o.values()
    ),
)


print("\n--- BLOCK CONTRACT ---")

for key, value in contract_checks_630v18o.items():
    print(
        f"{key:28s}:",
        value,
    )


print("\n--- SAFE CHECKPOINT ---")

print(
    "NPZ byte SHA exact                :",
    checkpoint_byte_checks_630v18o[
        "npz"
    ],
)

print(
    "JSON byte SHA exact               :",
    checkpoint_byte_checks_630v18o[
        "json"
    ],
)

print(
    "NPZ allow_pickle=False checks     :",
    npz_checks_630v18o,
)

print(
    "JSON checks                       :",
    json_checks_630v18o,
)

print(
    "Prior 18N round-trip all true     :",
    prior_roundtrip_all_true_630v18o,
)

print(
    "Dropout closure link exact        :",
    dropout_closure_link_exact_630v18o,
)


print("\n--- FINAL CLOSURE ---")

print(
    "Stuck family closed               :",
    FD004_STUCK_FAMILY_CLOSED_630V18O,
)

print(
    "Stuck final closure SHA-256       :",
    FD004_STUCK_FINAL_CLOSURE_SHA256_630V18O,
)


print(
    "\nPASS — FD004 stuck is evidence-locked using only resident "
    "authoritative hashes and direct recomputation. No stuck "
    "scenario was rerun."
)

print(
    "Next: close the complete 381-scenario label-free "
    "controlled-corruption execution across all five families."
)

In [ ]:
# STEP 6.30V-18P — FD004 COMPLETE 381-SCENARIO
# LABEL-FREE CONTROLLED-CORRUPTION EVIDENCE CLOSURE
#
# ZERO SCIENTIFIC EXECUTION.
#
# NO corruption.
# NO normalization/windows.
# NO deterministic inference.
# NO MC.
# NO reliability execution.
# NO DEV targets/errors.
# NO threshold/policy selection.
# NO TEST / official RUL.
#
# Purpose:
# 1. Verify all five family closures from resident objects.
# 2. Verify 102+51+102+63+63 = 381 executable scenarios.
# 3. Verify family manifests reconstruct the frozen 381-row execution manifest.
# 4. Verify executable 381 + inadmissible 60 = original 441 identities.
# 5. Freeze one master label-free corruption closure.
# 6. Save a small restart-safe JSON master checkpoint.
#
# IMPORTANT:
# No manually transcribed scientific-result hashes are used.

import pandas as pd
import numpy as np
import hashlib
import json
import os
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630v18p = (
    # Five family closures
    "all_checks_passed_630v18g",
    "FD004_BIAS_FAMILY_CLOSED_630V18G",
    "FD004_BIAS_FINAL_CLOSURE_630V18G",
    "FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G",

    "all_checks_passed_630v18ir1",
    "FD004_NOISE_FAMILY_CLOSED_630V18IR1",
    "FD004_NOISE_FINAL_CLOSURE_630V18IR1",
    "FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1",

    "all_checks_passed_630v18k",
    "FD004_DRIFT_FAMILY_CLOSED_630V18K",
    "FD004_DRIFT_FINAL_CLOSURE_630V18K",
    "FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K",

    "all_checks_passed_630v18m",
    "FD004_DROPOUT_FAMILY_CLOSED_630V18M",
    "FD004_DROPOUT_FINAL_CLOSURE_630V18M",
    "FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M",

    "all_checks_passed_630v18o",
    "FD004_STUCK_FAMILY_CLOSED_630V18O",
    "FD004_STUCK_FINAL_CLOSURE_630V18O",
    "FD004_STUCK_FINAL_CLOSURE_SHA256_630V18O",

    # Executed family manifests
    "FD004_BIAS_SCENARIO_MANIFEST_630V18E",
    "FD004_NOISE_SCENARIO_MANIFEST_630V18H",
    "FD004_DRIFT_SCENARIO_MANIFEST_630V18J",
    "FD004_DROPOUT_SCENARIO_MANIFEST_630V18L",
    "FD004_STUCK_SCENARIO_MANIFEST_630V18N",

    # Executed counts
    "FD004_BIAS_SCENARIOS_EXECUTED_630V18E",
    "FD004_NOISE_SCENARIOS_EXECUTED_630V18H",
    "FD004_DRIFT_SCENARIOS_EXECUTED_630V18J",
    "FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L",
    "FD004_STUCK_SCENARIOS_EXECUTED_630V18N",

    # Frozen scientific manifests
    "FD004_CORRUPTION_EXECUTION_MANIFEST_630U",
    "FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U",
    "FD004_CORRUPTION_MANIFEST_ANNOTATED_630U",

    # Admission authority
    "FD004_381_EXECUTION_ADMISSION_SHA256_630V18A",
)

missing_630v18p = [
    name
    for name in required_630v18p
    if name not in globals()
]

if missing_630v18p:
    raise RuntimeError(
        "Missing STEP 6.30V-18P authority object(s): "
        + ", ".join(missing_630v18p)
    )


# =============================================================================
# 1) ALL FIVE CLOSURE FLAGS MUST REMAIN PASSED
# =============================================================================

family_pass_flags_630v18p = {
    "bias":
        bool(all_checks_passed_630v18g)
        and
        bool(FD004_BIAS_FAMILY_CLOSED_630V18G),

    "noise":
        bool(all_checks_passed_630v18ir1)
        and
        bool(FD004_NOISE_FAMILY_CLOSED_630V18IR1),

    "drift":
        bool(all_checks_passed_630v18k)
        and
        bool(FD004_DRIFT_FAMILY_CLOSED_630V18K),

    "dropout":
        bool(all_checks_passed_630v18m)
        and
        bool(FD004_DROPOUT_FAMILY_CLOSED_630V18M),

    "stuck":
        bool(all_checks_passed_630v18o)
        and
        bool(FD004_STUCK_FAMILY_CLOSED_630V18O),
}

if not all(
    family_pass_flags_630v18p.values()
):
    failed_630v18p = [
        family
        for family, passed
        in family_pass_flags_630v18p.items()
        if not passed
    ]

    raise RuntimeError(
        "One or more family closures are not passed: "
        + ", ".join(failed_630v18p)
    )


# =============================================================================
# 2) RECOMPUTE EACH FAMILY CLOSURE SHA FROM ITS RESIDENT OBJECT
# =============================================================================

family_closure_objects_630v18p = {
    "bias":
        FD004_BIAS_FINAL_CLOSURE_630V18G,

    "noise":
        FD004_NOISE_FINAL_CLOSURE_630V18IR1,

    "drift":
        FD004_DRIFT_FINAL_CLOSURE_630V18K,

    "dropout":
        FD004_DROPOUT_FINAL_CLOSURE_630V18M,

    "stuck":
        FD004_STUCK_FINAL_CLOSURE_630V18O,
}


family_closure_sha_authorities_630v18p = {
    "bias":
        str(
            FD004_BIAS_FINAL_CLOSURE_SHA256_630V18G
        ),

    "noise":
        str(
            FD004_NOISE_FINAL_CLOSURE_SHA256_630V18IR1
        ),

    "drift":
        str(
            FD004_DRIFT_FINAL_CLOSURE_SHA256_630V18K
        ),

    "dropout":
        str(
            FD004_DROPOUT_FINAL_CLOSURE_SHA256_630V18M
        ),

    "stuck":
        str(
            FD004_STUCK_FINAL_CLOSURE_SHA256_630V18O
        ),
}


family_closure_sha_recomputed_630v18p = {
    family:
        hashlib.sha256(
            json.dumps(
                closure_object,
                sort_keys=True,
                separators=(",", ":"),
                default=str,
            ).encode("utf-8")
        ).hexdigest()

    for family, closure_object
    in family_closure_objects_630v18p.items()
}


family_closure_sha_checks_630v18p = {
    family:
        family_closure_sha_recomputed_630v18p[family]
        ==
        family_closure_sha_authorities_630v18p[family]

    for family
    in family_closure_objects_630v18p
}

if not all(
    family_closure_sha_checks_630v18p.values()
):
    failed_630v18p = [
        family
        for family, passed
        in family_closure_sha_checks_630v18p.items()
        if not passed
    ]

    raise RuntimeError(
        "Family closure SHA verification failed: "
        + ", ".join(failed_630v18p)
    )


# =============================================================================
# 3) FAMILY CLOSURE SEMANTICS
# =============================================================================

expected_family_counts_630v18p = {
    "bias": 102,
    "noise": 51,
    "drift": 102,
    "dropout": 63,
    "stuck": 63,
}

expected_family_status_630v18p = {
    "bias":
        "BIAS_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "noise":
        "NOISE_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "drift":
        "DRIFT_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "dropout":
        "DROPOUT_FAMILY_COMPLETE_EVIDENCE_LOCKED",

    "stuck":
        "STUCK_FAMILY_COMPLETE_EVIDENCE_LOCKED",
}


family_closure_semantic_checks_630v18p = {}

for family_630v18p, closure_630v18p in (
    family_closure_objects_630v18p.items()
):

    family_closure_semantic_checks_630v18p[
        family_630v18p
    ] = {
        "dataset":
            str(
                closure_630v18p.get(
                    "dataset"
                )
            )
            == "FD004",

        "family":
            str(
                closure_630v18p.get(
                    "family"
                )
            )
            == family_630v18p,

        "status":
            str(
                closure_630v18p.get(
                    "status"
                )
            )
            ==
            expected_family_status_630v18p[
                family_630v18p
            ],

        "scientific_scenario_count":
            int(
                closure_630v18p.get(
                    "scientific_scenario_count"
                )
            )
            ==
            expected_family_counts_630v18p[
                family_630v18p
            ],

        "full_mc_not_retained":
            bool(
                closure_630v18p.get(
                    "full_mc_matrices_retained"
                )
            )
            is False,

        "dev_outcomes_unused":
            bool(
                closure_630v18p.get(
                    "dev_targets_errors_used"
                )
            )
            is False,

        "threshold_unselected":
            bool(
                closure_630v18p.get(
                    "threshold_selected"
                )
            )
            is False,

        "policy_unselected":
            bool(
                closure_630v18p.get(
                    "policy_selected"
                )
            )
            is False,

        "test_unread":
            bool(
                closure_630v18p.get(
                    "test_contents_read"
                )
            )
            is False,

        "rul_unread":
            bool(
                closure_630v18p.get(
                    "official_rul_contents_read"
                )
            )
            is False,
    }


for family_630v18p, checks_630v18p in (
    family_closure_semantic_checks_630v18p.items()
):

    if not all(
        checks_630v18p.values()
    ):
        failed_630v18p = [
            key
            for key, passed
            in checks_630v18p.items()
            if not passed
        ]

        raise RuntimeError(
            f"{family_630v18p} closure semantic check failed: "
            + ", ".join(failed_630v18p)
        )


# =============================================================================
# 4) EXECUTED COUNTS
# =============================================================================

executed_counts_630v18p = {
    "bias":
        int(
            FD004_BIAS_SCENARIOS_EXECUTED_630V18E
        ),

    "noise":
        int(
            FD004_NOISE_SCENARIOS_EXECUTED_630V18H
        ),

    "drift":
        int(
            FD004_DRIFT_SCENARIOS_EXECUTED_630V18J
        ),

    "dropout":
        int(
            FD004_DROPOUT_SCENARIOS_EXECUTED_630V18L
        ),

    "stuck":
        int(
            FD004_STUCK_SCENARIOS_EXECUTED_630V18N
        ),
}


executed_count_checks_630v18p = {
    family:
        executed_counts_630v18p[family]
        ==
        expected_family_counts_630v18p[family]

    for family
    in expected_family_counts_630v18p
}

if not all(
    executed_count_checks_630v18p.values()
):
    raise RuntimeError(
        "One or more family execution counts changed."
    )


total_executed_630v18p = int(
    sum(
        executed_counts_630v18p.values()
    )
)

if total_executed_630v18p != 381:
    raise RuntimeError(
        "Executed family counts do not sum to 381."
    )


# =============================================================================
# 5) RECONSTRUCT 381-ROW EXECUTION MANIFEST FROM FAMILY MANIFESTS
# =============================================================================

family_manifests_630v18p = {
    "bias":
        FD004_BIAS_SCENARIO_MANIFEST_630V18E,

    "noise":
        FD004_NOISE_SCENARIO_MANIFEST_630V18H,

    "drift":
        FD004_DRIFT_SCENARIO_MANIFEST_630V18J,

    "dropout":
        FD004_DROPOUT_SCENARIO_MANIFEST_630V18L,

    "stuck":
        FD004_STUCK_SCENARIO_MANIFEST_630V18N,
}


for family_630v18p, manifest_630v18p in (
    family_manifests_630v18p.items()
):

    if len(
        manifest_630v18p
    ) != expected_family_counts_630v18p[
        family_630v18p
    ]:
        raise RuntimeError(
            f"{family_630v18p} manifest count changed."
        )

    if not np.all(
        manifest_630v18p[
            "family"
        ].astype(str).to_numpy()
        ==
        family_630v18p
    ):
        raise RuntimeError(
            f"{family_630v18p} manifest contains another family."
        )


combined_family_manifest_630v18p = (
    pd.concat(
        [
            family_manifests_630v18p[
                "bias"
            ],
            family_manifests_630v18p[
                "noise"
            ],
            family_manifests_630v18p[
                "drift"
            ],
            family_manifests_630v18p[
                "dropout"
            ],
            family_manifests_630v18p[
                "stuck"
            ],
        ],
        axis=0,
        ignore_index=True,
    )
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


frozen_execution_manifest_630v18p = (
    FD004_CORRUPTION_EXECUTION_MANIFEST_630U
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    combined_family_manifest_630v18p
) != 381:
    raise RuntimeError(
        "Combined family manifest does not have 381 rows."
    )

if len(
    frozen_execution_manifest_630v18p
) != 381:
    raise RuntimeError(
        "Frozen execution manifest no longer has 381 rows."
    )


manifest_columns_exact_630v18p = bool(
    tuple(
        combined_family_manifest_630v18p.columns
    )
    ==
    tuple(
        frozen_execution_manifest_630v18p.columns
    )
)

if not manifest_columns_exact_630v18p:
    raise RuntimeError(
        "Combined-family and frozen execution-manifest columns differ."
    )


execution_manifest_exact_630v18p = bool(
    combined_family_manifest_630v18p.equals(
        frozen_execution_manifest_630v18p
    )
)

if not execution_manifest_exact_630v18p:
    raise RuntimeError(
        "Five closed family manifests do not reconstruct "
        "the frozen 381-row execution manifest exactly."
    )


scenario_numbers_630v18p = (
    frozen_execution_manifest_630v18p[
        "scenario_number"
    ].to_numpy(dtype=np.int64)
)

scenario_ids_630v18p = (
    frozen_execution_manifest_630v18p[
        "scenario_id"
    ].astype(str).to_numpy()
)

execution_identity_checks_630v18p = {
    "unique_scenario_numbers":
        len(
            np.unique(
                scenario_numbers_630v18p
            )
        )
        == 381,

    "unique_scenario_ids":
        len(
            np.unique(
                scenario_ids_630v18p
            )
        )
        == 381,

    "strictly_increasing_numbers":
        bool(
            np.all(
                np.diff(
                    scenario_numbers_630v18p
                )
                > 0
            )
        ),
}

if not all(
    execution_identity_checks_630v18p.values()
):
    raise RuntimeError(
        "381-row execution identity check failed."
    )


# =============================================================================
# 6) VERIFY 381 EXECUTABLE + 60 INADMISSIBLE = ORIGINAL 441 IDENTITIES
# =============================================================================

inadmissible_manifest_630v18p = (
    FD004_CORRUPTION_INADMISSIBLE_MANIFEST_630U
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

annotated_manifest_630v18p = (
    FD004_CORRUPTION_MANIFEST_ANNOTATED_630U
    .copy()
    .sort_values(
        "scenario_number",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    inadmissible_manifest_630v18p
) != 60:
    raise RuntimeError(
        "Frozen structurally inadmissible manifest no longer has 60 rows."
    )

if len(
    annotated_manifest_630v18p
) != 441:
    raise RuntimeError(
        "Frozen annotated scientific manifest no longer has 441 rows."
    )


execution_keys_630v18p = set(
    zip(
        frozen_execution_manifest_630v18p[
            "scenario_number"
        ].astype(int),
        frozen_execution_manifest_630v18p[
            "scenario_id"
        ].astype(str),
    )
)

inadmissible_keys_630v18p = set(
    zip(
        inadmissible_manifest_630v18p[
            "scenario_number"
        ].astype(int),
        inadmissible_manifest_630v18p[
            "scenario_id"
        ].astype(str),
    )
)

annotated_keys_630v18p = set(
    zip(
        annotated_manifest_630v18p[
            "scenario_number"
        ].astype(int),
        annotated_manifest_630v18p[
            "scenario_id"
        ].astype(str),
    )
)


partition_checks_630v18p = {
    "execution_count_381":
        len(
            execution_keys_630v18p
        )
        == 381,

    "inadmissible_count_60":
        len(
            inadmissible_keys_630v18p
        )
        == 60,

    "scientific_count_441":
        len(
            annotated_keys_630v18p
        )
        == 441,

    "execution_inadmissible_disjoint":
        execution_keys_630v18p.isdisjoint(
            inadmissible_keys_630v18p
        ),

    "union_exact_scientific_manifest":
        (
            execution_keys_630v18p
            |
            inadmissible_keys_630v18p
        )
        ==
        annotated_keys_630v18p,
}

if not all(
    partition_checks_630v18p.values()
):
    failed_630v18p = [
        key
        for key, passed
        in partition_checks_630v18p.items()
        if not passed
    ]

    raise RuntimeError(
        "441-scenario executable/inadmissible partition failed: "
        + ", ".join(failed_630v18p)
    )


# =============================================================================
# 7) FAMILY POPULATION FROM FROZEN EXECUTION MANIFEST
# =============================================================================

observed_family_counts_630v18p = {
    str(family):
        int(count)

    for family, count
    in frozen_execution_manifest_630v18p[
        "family"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
    .items()
}


family_population_exact_630v18p = bool(
    observed_family_counts_630v18p
    ==
    expected_family_counts_630v18p
)

if not family_population_exact_630v18p:
    raise RuntimeError(
        "Frozen execution-manifest family population changed."
    )


# =============================================================================
# 8) FREEZE MASTER 381-SCENARIO LABEL-FREE CLOSURE
# =============================================================================

FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P = {
    "dataset":
        "FD004",

    "stage":
        "6.30V-18P",

    "status":
        "CONTROLLED_CORRUPTION_381_LABEL_FREE_COMPLETE_EVIDENCE_LOCKED",

    "scientific_manifest_count":
        441,

    "executable_scenario_count":
        381,

    "structurally_inadmissible_count":
        60,

    "family_counts":
        dict(
            expected_family_counts_630v18p
        ),

    "family_closure_sha256":
        dict(
            family_closure_sha_authorities_630v18p
        ),

    "execution_admission_sha256":
        str(
            FD004_381_EXECUTION_ADMISSION_SHA256_630V18A
        ),

    "family_manifests_reconstruct_execution_manifest":
        execution_manifest_exact_630v18p,

    "execution_inadmissible_partition_exact":
        bool(
            all(
                partition_checks_630v18p.values()
            )
        ),

    "all_family_closure_sha_checks":
        bool(
            all(
                family_closure_sha_checks_630v18p.values()
            )
        ),

    "all_family_semantic_checks":
        bool(
            all(
                all(
                    family_checks.values()
                )
                for family_checks
                in family_closure_semantic_checks_630v18p.values()
            )
        ),

    "full_mc_matrices_retained":
        False,

    "scientific_execution_performed_in_closure":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P = (
    hashlib.sha256(
        json.dumps(
            FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


# =============================================================================
# 9) SMALL RESTART-SAFE MASTER JSON CHECKPOINT
# =============================================================================

checkpoint_dir_630v18p = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
)

checkpoint_dir_630v18p.mkdir(
    parents=True,
    exist_ok=True,
)

master_json_path_630v18p = (
    checkpoint_dir_630v18p
    /
    "fd004_controlled_corruption_381_closure_630v18p_v1.json"
)

master_json_tmp_630v18p = (
    checkpoint_dir_630v18p
    /
    "fd004_controlled_corruption_381_closure_630v18p_v1.json.tmp"
)


master_checkpoint_payload_630v18p = {
    "closure":
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,

    "closure_sha256":
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P,
}


with open(
    master_json_tmp_630v18p,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        master_checkpoint_payload_630v18p,
        file,
        sort_keys=True,
        indent=2,
        default=str,
    )


os.replace(
    master_json_tmp_630v18p,
    master_json_path_630v18p,
)


def _file_sha256_630v18p(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:

        while True:
            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


master_json_file_sha_630v18p = (
    _file_sha256_630v18p(
        master_json_path_630v18p
    )
)


with open(
    master_json_path_630v18p,
    "r",
    encoding="utf-8",
) as file:

    master_json_loaded_630v18p = json.load(
        file
    )


master_checkpoint_roundtrip_exact_630v18p = bool(
    master_json_loaded_630v18p
    ==
    master_checkpoint_payload_630v18p
)

if not master_checkpoint_roundtrip_exact_630v18p:
    raise RuntimeError(
        "Master 381-scenario closure JSON failed round-trip verification."
    )


FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_JSON_PATH_630V18P = str(
    master_json_path_630v18p
)

FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_JSON_SHA256_630V18P = (
    master_json_file_sha_630v18p
)

FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_ROUNDTRIP_EXACT_630V18P = (
    master_checkpoint_roundtrip_exact_630v18p
)


# =============================================================================
# 10) FINAL FLAGS
# =============================================================================

FD004_CONTROLLED_CORRUPTION_381_COMPLETE_630V18P = True
FD004_CONTROLLED_CORRUPTION_LABEL_FREE_PHASE_CLOSED_630V18P = True

FD004_SCIENTIFIC_EXECUTION_PERFORMED_630V18P = False
FD004_DEV_TARGETS_ERRORS_USED_630V18P = False
FD004_THRESHOLD_SELECTED_630V18P = False
FD004_POLICY_SELECTED_630V18P = False
FD004_TEST_CONTENTS_READ_630V18P = False
FD004_RUL_CONTENTS_READ_630V18P = False

all_checks_passed_630v18p = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30V-18P — FD004 COMPLETE 381-SCENARIO "
    "LABEL-FREE CONTROLLED-CORRUPTION EVIDENCE CLOSURE"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("Corruption scenarios rerun        :", 0)
print("Model inference performed         :", False)
print("MC execution performed            :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- FAMILY CLOSURES ---")

for family_630v18p in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    print(
        f"{family_630v18p:8s} closure SHA exact        :",
        family_closure_sha_checks_630v18p[
            family_630v18p
        ],
    )

    print(
        f"{family_630v18p:8s} executed count           :",
        executed_counts_630v18p[
            family_630v18p
        ],
    )


print("\n--- 381-SCENARIO EXECUTION MANIFEST ---")

print(
    "Combined family rows              :",
    len(
        combined_family_manifest_630v18p
    ),
)

print(
    "Frozen execution-manifest rows    :",
    len(
        frozen_execution_manifest_630v18p
    ),
)

print(
    "Manifest columns exact            :",
    manifest_columns_exact_630v18p,
)

print(
    "Five manifests reconstruct 381    :",
    execution_manifest_exact_630v18p,
)

print(
    "Family populations exact          :",
    family_population_exact_630v18p,
)

print(
    "Execution identities exact        :",
    all(
        execution_identity_checks_630v18p.values()
    ),
)


print("\n--- ORIGINAL 441-SCENARIO SCIENTIFIC DESIGN ---")

print(
    "Scientific manifest rows          :",
    len(
        annotated_manifest_630v18p
    ),
)

print(
    "Executable rows                   :",
    len(
        frozen_execution_manifest_630v18p
    ),
)

print(
    "Structurally inadmissible rows     :",
    len(
        inadmissible_manifest_630v18p
    ),
)

print(
    "Executable/inadmissible disjoint  :",
    partition_checks_630v18p[
        "execution_inadmissible_disjoint"
    ],
)

print(
    "381 + 60 reconstructs all 441     :",
    partition_checks_630v18p[
        "union_exact_scientific_manifest"
    ],
)


print("\n--- MASTER CLOSURE ---")

print(
    "381 scenarios complete            :",
    FD004_CONTROLLED_CORRUPTION_381_COMPLETE_630V18P,
)

print(
    "Label-free corruption phase closed:",
    FD004_CONTROLLED_CORRUPTION_LABEL_FREE_PHASE_CLOSED_630V18P,
)

print(
    "Master closure SHA-256            :",
    FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P,
)


print("\n--- MASTER RESTART CHECKPOINT ---")

print(
    "JSON path                         :",
    FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_JSON_PATH_630V18P,
)

print(
    "JSON SHA-256                      :",
    FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_JSON_SHA256_630V18P,
)

print(
    "JSON round-trip exact             :",
    FD004_CONTROLLED_CORRUPTION_381_CHECKPOINT_ROUNDTRIP_EXACT_630V18P,
)


print(
    "\nPASS — all 381 structurally admissible FD004 controlled-corruption "
    "scenarios are complete and evidence-locked across all five families."
)

print(
    "The original 441-scenario scientific design is preserved: "
    "381 executed + 60 explicitly documented structurally inadmissible."
)

print(
    "No DEV outcomes/errors have been used, and TEST/RUL remain sealed."
)

print(
    "Next: begin the pre-specified DEV outcome/reporting stage for the "
    "frozen corruption evidence, still before any TEST access."
)

In [ ]:
# STEP 6.30W-0 — FD004 DEV OUTCOME-STAGE AUTHORITY / PROTOCOL LOCATOR
#
# ZERO DEV OUTCOME COMPUTATION.
#
# NO subtraction of target and prediction.
# NO MAE/RMSE/error computation.
# NO threshold selection.
# NO policy selection.
# NO TEST / official RUL access.
# NO corruption / normalization / inference / MC / reliability execution.
#
# Purpose:
# 1. Verify the frozen 381-scenario closure is still intact.
# 2. Locate the exact resident DEV target / clean prediction / clean-risk authorities.
# 3. Verify the five retained corruption result matrices are resident.
# 4. Locate resident historical/protocol source cells relevant to
#    DEV outcome reporting / threshold / selective-policy selection.
#
# This is a locator/audit only. It does not use DEV outcomes.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) MASTER-CLOSURE PRECONDITION
# =============================================================================

required_630w0 = (
    "all_checks_passed_630v18p",
    "FD004_CONTROLLED_CORRUPTION_381_COMPLETE_630V18P",
    "FD004_CONTROLLED_CORRUPTION_LABEL_FREE_PHASE_CLOSED_630V18P",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P",
)

missing_630w0 = [
    name
    for name in required_630w0
    if name not in globals()
]

if missing_630w0:
    raise RuntimeError(
        "Missing STEP 6.30W-0 authority object(s): "
        + ", ".join(missing_630w0)
    )

if not bool(all_checks_passed_630v18p):
    raise RuntimeError(
        "STEP 6.30V-18P must remain passed."
    )

if not bool(
    FD004_CONTROLLED_CORRUPTION_381_COMPLETE_630V18P
):
    raise RuntimeError(
        "381-scenario corruption phase is not marked complete."
    )

if not bool(
    FD004_CONTROLLED_CORRUPTION_LABEL_FREE_PHASE_CLOSED_630V18P
):
    raise RuntimeError(
        "Label-free corruption phase is not marked closed."
    )

master_closure_recomputed_sha_630w0 = hashlib.sha256(
    json.dumps(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

master_closure_exact_630w0 = bool(
    master_closure_recomputed_sha_630w0
    ==
    str(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P
    )
)

if not master_closure_exact_630w0:
    raise RuntimeError(
        "Frozen 381-scenario master closure changed."
    )


# =============================================================================
# 1) SAFE RESIDENT NUMERIC-OBJECT LOCATOR
# =============================================================================

def _numeric_vector_info_630w0(name, value):
    try:
        if isinstance(value, pd.Series):
            arr = value.to_numpy()

        elif isinstance(value, np.ndarray):
            arr = value

        else:
            return None

        arr = np.asarray(arr)

        if arr.ndim != 1:
            return None

        if arr.size != 10824:
            return None

        if not np.issubdtype(
            arr.dtype,
            np.number,
        ):
            return None

        return {
            "name": name,
            "type": type(value).__name__,
            "shape": tuple(arr.shape),
            "dtype": str(arr.dtype),
            "finite": bool(
                np.isfinite(
                    arr.astype(
                        np.float64,
                        copy=False,
                    )
                ).all()
            ),
        }

    except Exception:
        return None


vector_candidates_630w0 = []

for name_630w0, value_630w0 in list(
    globals().items()
):
    info_630w0 = _numeric_vector_info_630w0(
        name_630w0,
        value_630w0,
    )

    if info_630w0 is not None:
        vector_candidates_630w0.append(
            info_630w0
        )


def _rank_target_name_630w0(name):
    upper = name.upper()
    score = 0

    if "FD004" in upper:
        score += 8
    if "DEV" in upper:
        score += 8
    if upper.startswith("Y_") or "_Y_" in upper:
        score += 5
    if "TARGET" in upper:
        score += 5
    if "RUL" in upper:
        score += 3
    if "630G" in upper:
        score += 4

    return (-score, name)


def _rank_prediction_name_630w0(name):
    upper = name.upper()
    score = 0

    if "FD004" in upper:
        score += 8
    if "DEV" in upper:
        score += 6
    if "PRED" in upper:
        score += 7
    if "CLEAN" in upper:
        score += 5
    if "BEST" in upper:
        score += 3
    if "630J" in upper or "630K" in upper:
        score += 4

    return (-score, name)


def _rank_risk_name_630w0(name):
    upper = name.upper()
    score = 0

    if "FD004" in upper:
        score += 8
    if "DEV" in upper:
        score += 6
    if "RISK" in upper:
        score += 8
    if "FINAL" in upper:
        score += 5
    if "630R" in upper:
        score += 4
    if "CLEAN" in upper:
        score += 3

    return (-score, name)


target_candidates_630w0 = sorted(
    vector_candidates_630w0,
    key=lambda row: _rank_target_name_630w0(
        row["name"]
    ),
)[:20]

prediction_candidates_630w0 = sorted(
    vector_candidates_630w0,
    key=lambda row: _rank_prediction_name_630w0(
        row["name"]
    ),
)[:20]

risk_candidates_630w0 = sorted(
    [
        row
        for row in vector_candidates_630w0
        if "RISK" in row["name"].upper()
        or "630R" in row["name"].upper()
    ],
    key=lambda row: _rank_risk_name_630w0(
        row["name"]
    ),
)[:20]


# =============================================================================
# 2) KNOWN CORRUPTION-FAMILY RETAINED MATRIX AUTHORITIES
# =============================================================================

family_matrix_authorities_630w0 = {
    "bias": (
        "FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E",
        "FD004_BIAS_MC_STD_630V18E",
        "FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E",
        102,
    ),

    "noise": (
        "FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H",
        "FD004_NOISE_MC_STD_630V18H",
        "FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H",
        51,
    ),

    "drift": (
        "FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J",
        "FD004_DRIFT_MC_STD_630V18J",
        "FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J",
        102,
    ),

    "dropout": (
        "FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L",
        "FD004_DROPOUT_MC_STD_630V18L",
        "FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L",
        63,
    ),

    "stuck": (
        "FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N",
        "FD004_STUCK_MC_STD_630V18N",
        "FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N",
        63,
    ),
}


family_matrix_checks_630w0 = {}

for (
    family_630w0,
    (
        pred_name_630w0,
        std_name_630w0,
        risk_name_630w0,
        expected_rows_630w0,
    ),
) in family_matrix_authorities_630w0.items():

    entry_630w0 = {
        "prediction_name": pred_name_630w0,
        "mc_std_name": std_name_630w0,
        "risk_name": risk_name_630w0,

        "prediction_present":
            pred_name_630w0 in globals(),

        "mc_std_present":
            std_name_630w0 in globals(),

        "risk_present":
            risk_name_630w0 in globals(),
    }

    if all(
        (
            entry_630w0["prediction_present"],
            entry_630w0["mc_std_present"],
            entry_630w0["risk_present"],
        )
    ):
        pred_matrix_630w0 = np.asarray(
            globals()[pred_name_630w0]
        )

        std_matrix_630w0 = np.asarray(
            globals()[std_name_630w0]
        )

        risk_matrix_630w0 = np.asarray(
            globals()[risk_name_630w0]
        )

        entry_630w0.update(
            {
                "prediction_shape":
                    tuple(pred_matrix_630w0.shape),

                "mc_std_shape":
                    tuple(std_matrix_630w0.shape),

                "risk_shape":
                    tuple(risk_matrix_630w0.shape),

                "shapes_exact":
                    (
                        pred_matrix_630w0.shape
                        ==
                        (expected_rows_630w0, 10824)
                        and
                        std_matrix_630w0.shape
                        ==
                        (expected_rows_630w0, 10824)
                        and
                        risk_matrix_630w0.shape
                        ==
                        (expected_rows_630w0, 10824)
                    ),

                "all_finite":
                    bool(
                        np.isfinite(
                            pred_matrix_630w0
                        ).all()
                        and
                        np.isfinite(
                            std_matrix_630w0
                        ).all()
                        and
                        np.isfinite(
                            risk_matrix_630w0
                        ).all()
                    ),
            }
        )

    family_matrix_checks_630w0[
        family_630w0
    ] = entry_630w0


if not all(
    entry.get(
        "shapes_exact",
        False,
    )
    and
    entry.get(
        "all_finite",
        False,
    )
    for entry
    in family_matrix_checks_630w0.values()
):
    raise RuntimeError(
        "One or more frozen corruption-family retained matrices "
        "are missing or have changed shape/validity."
    )


# =============================================================================
# 3) LOCATE RESIDENT CLEAN-DEV / OUTCOME-STAGE NAMED AUTHORITIES
#
# Names/types only. No outcome arithmetic.
# =============================================================================

authority_name_tokens_630w0 = (
    "DEV",
    "TARGET",
    "PRED",
    "RISK",
    "THRESH",
    "COVERAGE",
    "POLICY",
    "SELECTIVE",
    "OUTCOME",
    "ERROR",
    "MAE",
)

named_authorities_630w0 = []

for name_630w0, value_630w0 in list(
    globals().items()
):
    upper_630w0 = name_630w0.upper()

    if "FD004" not in upper_630w0:
        continue

    if not any(
        token in upper_630w0
        for token
        in authority_name_tokens_630w0
    ):
        continue

    type_name_630w0 = type(
        value_630w0
    ).__name__

    shape_630w0 = None

    if isinstance(
        value_630w0,
        np.ndarray,
    ):
        shape_630w0 = tuple(
            value_630w0.shape
        )

    elif isinstance(
        value_630w0,
        pd.Series,
    ):
        shape_630w0 = tuple(
            value_630w0.shape
        )

    elif isinstance(
        value_630w0,
        pd.DataFrame,
    ):
        shape_630w0 = tuple(
            value_630w0.shape
        )

    named_authorities_630w0.append(
        {
            "name":
                name_630w0,

            "type":
                type_name_630w0,

            "shape":
                shape_630w0,
        }
    )

named_authorities_630w0 = sorted(
    named_authorities_630w0,
    key=lambda row: row["name"],
)


# =============================================================================
# 4) STATIC LOCATOR FOR HISTORICAL / PRE-SPECIFIED OUTCOME-POLICY SOURCE
#
# Search resident notebook-source strings only.
# Nothing is executed.
# =============================================================================

source_records_630w0 = []
seen_source_records_630w0 = set()


def _consider_source_630w0(
    origin_630w0,
    sequence_630w0,
    text_630w0,
):
    if not isinstance(
        text_630w0,
        str,
    ):
        return

    if len(text_630w0) < 100:
        return

    lowered_630w0 = text_630w0.lower()

    keyword_flags_630w0 = {
        "dev":
            "dev" in lowered_630w0,

        "threshold":
            "threshold" in lowered_630w0,

        "coverage":
            "coverage" in lowered_630w0,

        "selective":
            "selective" in lowered_630w0,

        "mae":
            "mae" in lowered_630w0,

        "policy":
            "policy" in lowered_630w0,

        "test":
            "test" in lowered_630w0,
    }

    score_630w0 = sum(
        int(
            keyword_flags_630w0[key]
        )
        for key in (
            "dev",
            "threshold",
            "coverage",
            "selective",
            "mae",
            "policy",
        )
    )

    if score_630w0 < 4:
        return

    sha_630w0 = hashlib.sha256(
        text_630w0.encode("utf-8")
    ).hexdigest()

    dedupe_key_630w0 = (
        sha_630w0,
        origin_630w0,
        sequence_630w0,
    )

    if dedupe_key_630w0 in seen_source_records_630w0:
        return

    seen_source_records_630w0.add(
        dedupe_key_630w0
    )

    matching_lines_630w0 = []

    for line_no_630w0, line_630w0 in enumerate(
        text_630w0.splitlines(),
        start=1,
    ):
        line_lower_630w0 = line_630w0.lower()

        if any(
            token in line_lower_630w0
            for token in (
                "threshold",
                "coverage",
                "selective",
                "mae",
                "policy",
            )
        ):
            matching_lines_630w0.append(
                (
                    line_no_630w0,
                    line_630w0.strip(),
                )
            )

        if len(matching_lines_630w0) >= 8:
            break

    source_records_630w0.append(
        {
            "origin":
                origin_630w0,

            "sequence":
                sequence_630w0,

            "sha256":
                sha_630w0,

            "length":
                len(text_630w0),

            "score":
                score_630w0,

            "flags":
                keyword_flags_630w0,

            "matching_lines":
                tuple(
                    matching_lines_630w0
                ),
        }
    )


# Notebook input history, if resident.
if "In" in globals():
    try:
        for idx_630w0, text_630w0 in enumerate(
            In
        ):
            _consider_source_630w0(
                "In",
                idx_630w0,
                text_630w0,
            )
    except Exception:
        pass


# Resident source-like string objects.
for name_630w0, value_630w0 in list(
    globals().items()
):
    if not isinstance(
        value_630w0,
        str,
    ):
        continue

    upper_630w0 = name_630w0.upper()

    if not any(
        token in upper_630w0
        for token in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "PROTOCOL",
            "CONTRACT",
        )
    ):
        continue

    _consider_source_630w0(
        "global:" + name_630w0,
        None,
        value_630w0,
    )


source_records_630w0 = sorted(
    source_records_630w0,
    key=lambda row: (
        -row["score"],
        row["origin"],
        -1
        if row["sequence"] is None
        else row["sequence"],
    ),
)


# =============================================================================
# 5) LOCK AUDIT
# =============================================================================

FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_630W0 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-0",

    "status":
        "DEV_OUTCOME_STAGE_AUTHORITY_LOCATED_NO_OUTCOMES_COMPUTED",

    "master_corruption_closure_exact":
        master_closure_exact_630w0,

    "resident_vector_candidate_count":
        len(vector_candidates_630w0),

    "target_candidate_names":
        tuple(
            row["name"]
            for row
            in target_candidates_630w0
        ),

    "prediction_candidate_names":
        tuple(
            row["name"]
            for row
            in prediction_candidates_630w0
        ),

    "risk_candidate_names":
        tuple(
            row["name"]
            for row
            in risk_candidates_630w0
        ),

    "family_matrix_checks":
        family_matrix_checks_630w0,

    "protocol_source_candidate_count":
        len(source_records_630w0),

    "dev_outcomes_computed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_SHA256_630W0 = (
    hashlib.sha256(
        json.dumps(
            FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_630W0,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w0 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-0 — FD004 DEV OUTCOME-STAGE "
    "AUTHORITY / PROTOCOL LOCATOR"
)

print("=" * 176)

print("DEV outcome computation performed :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "381-scenario master closure exact :",
    master_closure_exact_630w0,
)


print(
    "\n--- 10824-LENGTH NUMERIC VECTOR CANDIDATES: TARGET-RANKED ---"
)

for row_630w0 in target_candidates_630w0:
    print(
        f"{row_630w0['name']:<58s} "
        f"type={row_630w0['type']:<10s} "
        f"dtype={row_630w0['dtype']:<10s} "
        f"finite={row_630w0['finite']}"
    )


print(
    "\n--- 10824-LENGTH NUMERIC VECTOR CANDIDATES: PREDICTION-RANKED ---"
)

for row_630w0 in prediction_candidates_630w0:
    print(
        f"{row_630w0['name']:<58s} "
        f"type={row_630w0['type']:<10s} "
        f"dtype={row_630w0['dtype']:<10s} "
        f"finite={row_630w0['finite']}"
    )


print(
    "\n--- 10824-LENGTH NUMERIC VECTOR CANDIDATES: RISK-RANKED ---"
)

if risk_candidates_630w0:
    for row_630w0 in risk_candidates_630w0:
        print(
            f"{row_630w0['name']:<58s} "
            f"type={row_630w0['type']:<10s} "
            f"dtype={row_630w0['dtype']:<10s} "
            f"finite={row_630w0['finite']}"
        )
else:
    print("None located.")


print(
    "\n--- FROZEN CORRUPTION FAMILY RESULT MATRICES ---"
)

for (
    family_630w0,
    entry_630w0,
) in family_matrix_checks_630w0.items():

    print(
        f"{family_630w0:8s}: "
        f"pred={entry_630w0.get('prediction_shape')} "
        f"std={entry_630w0.get('mc_std_shape')} "
        f"risk={entry_630w0.get('risk_shape')} "
        f"exact={entry_630w0.get('shapes_exact')} "
        f"finite={entry_630w0.get('all_finite')}"
    )


print(
    "\n--- FD004 NAMED DEV / OUTCOME / POLICY AUTHORITIES ---"
)

for row_630w0 in named_authorities_630w0:
    print(
        f"{row_630w0['name']:<70s} "
        f"type={row_630w0['type']:<14s} "
        f"shape={row_630w0['shape']}"
    )


print(
    "\n--- HISTORICAL / PRE-SPECIFIED OUTCOME-POLICY SOURCE CANDIDATES ---"
)

print(
    "Candidate count                   :",
    len(source_records_630w0),
)

for idx_630w0, record_630w0 in enumerate(
    source_records_630w0[:20],
    start=1,
):
    print(
        f"\nSOURCE CANDIDATE {idx_630w0}"
    )

    print(
        "  origin                          :",
        record_630w0["origin"],
    )

    print(
        "  sequence                        :",
        record_630w0["sequence"],
    )

    print(
        "  SHA-256                         :",
        record_630w0["sha256"],
    )

    print(
        "  length                          :",
        record_630w0["length"],
    )

    print(
        "  keyword score                   :",
        record_630w0["score"],
    )

    print(
        "  flags                           :",
        record_630w0["flags"],
    )

    print(
        "  first matching lines            :"
    )

    for (
        line_no_630w0,
        line_630w0,
    ) in record_630w0["matching_lines"]:
        print(
            f"    {line_no_630w0:05d} | "
            f"{line_630w0[:220]}"
        )


print(
    "\n--- AUDIT FINGERPRINT ---"
)

print(
    "Authority/protocol audit SHA-256  :",
    FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_SHA256_630W0,
)

print(
    "\nPASS — the label-free corruption phase remains frozen. "
    "DEV outcome authorities and historical/pre-specified policy-source "
    "candidates were only located; no DEV error or threshold was computed."
)

print(
    "Next: bind the exact DEV target, clean prediction/risk, and "
    "the governing outcome/threshold protocol from this printed evidence "
    "before computing any DEV outcome."
)

In [ ]:
# STEP 6.30W-1 — FD004 HISTORICAL DEV THRESHOLD / POLICY SOURCE RECOVERY
#
# STATIC SOURCE AUDIT ONLY.
# NO DEV outcome arithmetic.
# NO threshold/policy selection.
# NO corruption / inference / MC / reliability execution.
# NO TEST / official RUL access.

import ast
import re
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w1 = (
    "all_checks_passed_630w0",
    "FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_630W0",
    "FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_SHA256_630W0",
)

missing_630w1 = [
    name
    for name in required_630w1
    if name not in globals()
]

if missing_630w1:
    raise RuntimeError(
        "Missing STEP 6.30W-1 authority object(s): "
        + ", ".join(missing_630w1)
    )

if not bool(all_checks_passed_630w0):
    raise RuntimeError(
        "STEP 6.30W-0 must remain passed."
    )

w0_recomputed_sha_630w1 = hashlib.sha256(
    json.dumps(
        FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_630W0,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    w0_recomputed_sha_630w1
    !=
    str(FD004_DEV_OUTCOME_STAGE_AUTHORITY_AUDIT_SHA256_630W0)
):
    raise RuntimeError(
        "STEP 6.30W-0 audit object changed."
    )


# =============================================================================
# 1) BUILD DEDUPLICATED SOURCE CORPUS
# =============================================================================

source_corpus_630w1 = []
seen_sha_630w1 = set()


def _add_source_630w1(origin, sequence, text):
    if not isinstance(text, str):
        return

    if len(text) < 100:
        return

    sha = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if sha in seen_sha_630w1:
        return

    seen_sha_630w1.add(sha)

    source_corpus_630w1.append(
        {
            "origin": origin,
            "sequence": sequence,
            "sha256": sha,
            "text": text,
        }
    )


if "In" in globals():
    try:
        for idx_630w1, text_630w1 in enumerate(In):
            _add_source_630w1(
                "In",
                idx_630w1,
                text_630w1,
            )
    except Exception:
        pass


for name_630w1, value_630w1 in list(globals().items()):
    if not isinstance(value_630w1, str):
        continue

    upper_630w1 = name_630w1.upper()

    if any(
        token in upper_630w1
        for token in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "PROTOCOL",
            "CONTRACT",
            "NOTEBOOK",
        )
    ):
        _add_source_630w1(
            "global:" + name_630w1,
            None,
            value_630w1,
        )


# =============================================================================
# 2) STATIC SELECTION-SIGNAL SCORING
# =============================================================================

strong_patterns_630w1 = (
    r"\baccepted_mask\b",
    r"\babstain(?:ed|ing)?\b",
    r"\bselective[_ ]?mae\b",
    r"\bcoverage\b",
    r"\bcandidate[_ ]?threshold",
    r"\bthreshold[_ ]?candidate",
    r"\brisk\s*<=",
    r"\brisk\s*<",
    r"\bquantile\s*\(",
    r"\bpercentile\s*\(",
    r"\bargmin\s*\(",
    r"\bargmax\s*\(",
    r"\bselective[_ ]?gain\b",
    r"\brelative[_ ]?reduction\b",
    r"\baccepted[_ ]?mae\b",
    r"\babstained[_ ]?mae\b",
)

medium_patterns_630w1 = (
    r"\bthreshold\b",
    r"\bpolicy\b",
    r"\baccept(?:ed|ance)?\b",
    r"\bcoverage\b",
    r"\bmae\b",
    r"\brisk\b",
)


def _selection_score_630w1(text):
    lowered = text.lower()

    strong_hits = sum(
        len(
            re.findall(
                pattern,
                lowered,
                flags=re.IGNORECASE,
            )
        )
        for pattern in strong_patterns_630w1
    )

    medium_hits = sum(
        len(
            re.findall(
                pattern,
                lowered,
                flags=re.IGNORECASE,
            )
        )
        for pattern in medium_patterns_630w1
    )

    score = (
        5 * strong_hits
        +
        min(
            medium_hits,
            20,
        )
    )

    guard_only_hits = len(
        re.findall(
            r"threshold_selected[^=\n]*=\s*false|"
            r"policy_selected[^=\n]*=\s*false|"
            r"no threshold|"
            r"no policy",
            lowered,
            flags=re.IGNORECASE,
        )
    )

    score -= min(
        guard_only_hits,
        20,
    )

    return (
        score,
        strong_hits,
        medium_hits,
        guard_only_hits,
    )


# =============================================================================
# 3) AST AUDIT FOR REAL THRESHOLD/POLICY ASSIGNMENTS
# =============================================================================

def _node_source_630w1(text, node):
    try:
        return ast.get_source_segment(
            text,
            node,
        )
    except Exception:
        return None


def _target_names_630w1(target):
    names = []

    if isinstance(target, ast.Name):
        names.append(target.id)

    elif isinstance(
        target,
        (ast.Tuple, ast.List),
    ):
        for elt in target.elts:
            names.extend(
                _target_names_630w1(
                    elt
                )
            )

    return names


def _is_false_or_none_literal_630w1(node):
    return (
        isinstance(node, ast.Constant)
        and
        node.value in (
            False,
            None,
        )
    )


def _ast_selection_evidence_630w1(text):
    assignments = []
    function_defs = []

    try:
        tree = ast.parse(text)
    except Exception:
        return assignments, function_defs

    for node in ast.walk(tree):

        if isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef),
        ):
            lower_name = node.name.lower()

            if any(
                token in lower_name
                for token in (
                    "threshold",
                    "policy",
                    "selective",
                    "coverage",
                    "accept",
                    "abstain",
                )
            ):
                function_defs.append(
                    {
                        "name":
                            node.name,

                        "line":
                            int(
                                getattr(
                                    node,
                                    "lineno",
                                    -1,
                                )
                            ),

                        "source":
                            _node_source_630w1(
                                text,
                                node,
                            ),
                    }
                )

        if isinstance(
            node,
            (ast.Assign, ast.AnnAssign),
        ):
            if isinstance(node, ast.Assign):
                targets = []

                for target in node.targets:
                    targets.extend(
                        _target_names_630w1(
                            target
                        )
                    )

                value_node = node.value

            else:
                targets = _target_names_630w1(
                    node.target
                )
                value_node = node.value

            for target_name in targets:
                lower_target = target_name.lower()

                if not any(
                    token in lower_target
                    for token in (
                        "threshold",
                        "coverage",
                        "selective",
                        "accept",
                        "abstain",
                        "policy",
                    )
                ):
                    continue

                if (
                    value_node is not None
                    and
                    _is_false_or_none_literal_630w1(
                        value_node
                    )
                ):
                    continue

                assignments.append(
                    {
                        "target":
                            target_name,

                        "line":
                            int(
                                getattr(
                                    node,
                                    "lineno",
                                    -1,
                                )
                            ),

                        "source":
                            _node_source_630w1(
                                text,
                                node,
                            ),
                    }
                )

    return assignments, function_defs


# =============================================================================
# 4) EXTRACT COMPACT LINE CONTEXTS
# =============================================================================

line_signal_regex_630w1 = re.compile(
    r"accepted_mask|"
    r"abstain|"
    r"selective[_ ]?mae|"
    r"coverage|"
    r"candidate[_ ]?threshold|"
    r"threshold[_ ]?candidate|"
    r"risk\s*<=|"
    r"risk\s*<|"
    r"quantile\s*\(|"
    r"percentile\s*\(|"
    r"argmin\s*\(|"
    r"argmax\s*\(|"
    r"selective[_ ]?gain|"
    r"relative[_ ]?reduction|"
    r"accepted[_ ]?mae|"
    r"abstained[_ ]?mae",
    flags=re.IGNORECASE,
)


def _line_contexts_630w1(text):
    lines = text.splitlines()

    matched_lines = [
        idx
        for idx, line in enumerate(
            lines,
            start=1,
        )
        if line_signal_regex_630w1.search(
            line
        )
    ]

    contexts = []

    for line_no in matched_lines[:12]:
        start = max(
            1,
            line_no - 3,
        )

        end = min(
            len(lines),
            line_no + 3,
        )

        snippet = tuple(
            (
                idx,
                lines[
                    idx - 1
                ],
            )
            for idx in range(
                start,
                end + 1,
            )
        )

        contexts.append(
            {
                "match_line":
                    line_no,

                "snippet":
                    snippet,
            }
        )

    return contexts


# =============================================================================
# 5) RANK TRUE CANDIDATES
# =============================================================================

candidates_630w1 = []

for record_630w1 in source_corpus_630w1:
    text_630w1 = record_630w1[
        "text"
    ]

    (
        score_630w1,
        strong_hits_630w1,
        medium_hits_630w1,
        guard_hits_630w1,
    ) = _selection_score_630w1(
        text_630w1
    )

    (
        assignments_630w1,
        functions_630w1,
    ) = _ast_selection_evidence_630w1(
        text_630w1
    )

    score_630w1 += (
        12
        *
        len(
            assignments_630w1
        )
    )

    score_630w1 += (
        15
        *
        len(
            functions_630w1
        )
    )

    contexts_630w1 = _line_contexts_630w1(
        text_630w1
    )

    if (
        score_630w1 <= 0
        and
        not assignments_630w1
        and
        not functions_630w1
        and
        not contexts_630w1
    ):
        continue

    candidates_630w1.append(
        {
            "origin":
                record_630w1[
                    "origin"
                ],

            "sequence":
                record_630w1[
                    "sequence"
                ],

            "sha256":
                record_630w1[
                    "sha256"
                ],

            "length":
                len(
                    text_630w1
                ),

            "score":
                int(
                    score_630w1
                ),

            "strong_hits":
                int(
                    strong_hits_630w1
                ),

            "medium_hits":
                int(
                    medium_hits_630w1
                ),

            "guard_hits":
                int(
                    guard_hits_630w1
                ),

            "assignments":
                tuple(
                    assignments_630w1
                ),

            "functions":
                tuple(
                    functions_630w1
                ),

            "contexts":
                tuple(
                    contexts_630w1
                ),
        }
    )


candidates_630w1 = sorted(
    candidates_630w1,
    key=lambda row: (
        -row["score"],
        -len(
            row["assignments"]
        ),
        -len(
            row["functions"]
        ),
        -row["strong_hits"],
        row["origin"],
        -1
        if row["sequence"] is None
        else row["sequence"],
    ),
)


top_candidates_630w1 = tuple(
    candidates_630w1[:15]
)


# =============================================================================
# 6) LOCK SOURCE-RECOVERY AUDIT
# =============================================================================

audit_payload_630w1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-1",

    "status":
        "HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_RECOVERED_STATIC_ONLY",

    "w0_audit_exact":
        True,

    "source_corpus_count":
        len(
            source_corpus_630w1
        ),

    "candidate_count":
        len(
            candidates_630w1
        ),

    "top_candidates":
        tuple(
            {
                "origin":
                    row["origin"],

                "sequence":
                    row["sequence"],

                "sha256":
                    row["sha256"],

                "length":
                    row["length"],

                "score":
                    row["score"],

                "strong_hits":
                    row["strong_hits"],

                "assignment_targets":
                    tuple(
                        item["target"]
                        for item
                        in row["assignments"]
                    ),

                "function_names":
                    tuple(
                        item["name"]
                        for item
                        in row["functions"]
                    ),

                "context_match_lines":
                    tuple(
                        item[
                            "match_line"
                        ]
                        for item
                        in row["contexts"]
                    ),
            }
            for row
            in top_candidates_630w1
        ),

    "dev_outcomes_computed":
        False,

    "dev_targets_errors_used":
        False,

    "threshold_selected":
        False,

    "policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_630W1 = (
    audit_payload_630w1
)

FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_SHA256_630W1 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630w1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w1 = True


# =============================================================================
# 7) REPORT — COMPACT, ARRANGED OUTPUT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-1 — FD004 HISTORICAL DEV "
    "THRESHOLD / POLICY SOURCE RECOVERY"
)

print("=" * 176)

print("DEV outcome computation performed :", False)
print("DEV targets/errors used           :", False)
print("Threshold selected                :", False)
print("Policy selected                   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nSource corpus count                :",
    len(
        source_corpus_630w1
    ),
)

print(
    "Selection-source candidates       :",
    len(
        candidates_630w1
    ),
)


print(
    "\n--- TOP HISTORICAL / PRE-SPECIFIED SELECTION SOURCES ---"
)

for idx_630w1, row_630w1 in enumerate(
    top_candidates_630w1,
    start=1,
):
    print(
        f"\nCANDIDATE {idx_630w1}"
    )

    print(
        "  origin                          :",
        row_630w1[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        row_630w1[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        row_630w1[
            "sha256"
        ],
    )

    print(
        "  score / strong hits             :",
        (
            row_630w1[
                "score"
            ],
            row_630w1[
                "strong_hits"
            ],
        ),
    )

    print(
        "  real assignment targets         :",
        tuple(
            item[
                "target"
            ]
            for item
            in row_630w1[
                "assignments"
            ][:20]
        ),
    )

    print(
        "  selection-related functions     :",
        tuple(
            item[
                "name"
            ]
            for item
            in row_630w1[
                "functions"
            ][:20]
        ),
    )

    if row_630w1[
        "assignments"
    ]:
        print(
            "  exact assignment evidence:"
        )

        for item_630w1 in row_630w1[
            "assignments"
        ][:8]:
            print(
                f"    line {item_630w1['line']}: "
                f"{item_630w1['source']}"
            )

    if row_630w1[
        "functions"
    ]:
        print(
            "  exact function-definition evidence:"
        )

        for item_630w1 in row_630w1[
            "functions"
        ][:5]:
            source_head_630w1 = (
                item_630w1[
                    "source"
                ]
                or
                ""
            ).splitlines()

            source_head_630w1 = (
                source_head_630w1[0]
                if source_head_630w1
                else
                ""
            )

            print(
                f"    line {item_630w1['line']}: "
                f"{source_head_630w1}"
            )

    if row_630w1[
        "contexts"
    ]:
        print(
            "  compact source contexts:"
        )

        for context_630w1 in row_630w1[
            "contexts"
        ][:6]:

            print(
                f"    [match line "
                f"{context_630w1['match_line']}]"
            )

            for (
                line_no_630w1,
                line_630w1,
            ) in context_630w1[
                "snippet"
            ]:
                print(
                    f"      {line_no_630w1:05d} | "
                    f"{line_630w1[:220]}"
                )


print(
    "\n--- AUDIT FINGERPRINT ---"
)

print(
    "Threshold/policy source audit SHA :",
    FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_SHA256_630W1,
)

print(
    "\nPASS — historical/pre-specified threshold-policy source "
    "candidates were recovered statically. No DEV outcome, "
    "threshold, policy, TEST, or official RUL was used."
)

print(
    "Next: identify the single governing selection rule from these "
    "exact source contexts, then bind it before any DEV error computation."
)

In [ ]:
# STEP 6.30W-2 — FD004 GOVERNING THRESHOLD/POLICY METHODOLOGY RECOVERY
#
# STATIC / CONTRACT INSPECTION ONLY.
# NO FD004 DEV outcome arithmetic.
# NO threshold/policy selection.
# NO TEST / official RUL access.
#
# Prior-dataset NUMERIC results are not adopted here.
# This step recovers methodology structure only.

import ast
import re
import hashlib
import json
import numbers

required_630w2 = (
    "all_checks_passed_630w1",
    "FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_630W1",
    "FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_SHA256_630W1",
)

missing_630w2 = [
    n
    for n in required_630w2
    if n not in globals()
]

if missing_630w2:
    raise RuntimeError(
        "Missing STEP 6.30W-2 authority object(s): "
        + ", ".join(missing_630w2)
    )

if not bool(all_checks_passed_630w1):
    raise RuntimeError(
        "STEP 6.30W-1 must remain passed."
    )

w1_sha_630w2 = hashlib.sha256(
    json.dumps(
        FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_630W1,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    w1_sha_630w2
    !=
    str(
        FD004_HISTORICAL_DEV_THRESHOLD_POLICY_SOURCE_AUDIT_SHA256_630W1
    )
):
    raise RuntimeError(
        "STEP 6.30W-1 audit object changed."
    )


# =============================================================================
# 1) INSPECT PRIOR-DATASET POLICY CONTRACTS — NUMERICS REDACTED
# =============================================================================

contract_names_630w2 = (
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
)


def _redact_prior_numeric_630w2(value):
    if isinstance(value, bool) or value is None:
        return value

    if isinstance(value, numbers.Number):
        return "<PRIOR_DATASET_NUMERIC_REDACTED>"

    if isinstance(value, str):
        return value

    if isinstance(value, dict):
        return {
            str(k):
                _redact_prior_numeric_630w2(v)
            for k, v
            in value.items()
        }

    if isinstance(value, (tuple, list)):
        return type(value)(
            _redact_prior_numeric_630w2(v)
            for v in value
        )

    return f"<{type(value).__name__}>"


resident_contract_audit_630w2 = {}

for name_630w2 in contract_names_630w2:

    present_630w2 = (
        name_630w2 in globals()
    )

    value_630w2 = globals().get(
        name_630w2
    )

    resident_contract_audit_630w2[
        name_630w2
    ] = {
        "present":
            bool(present_630w2),

        "type":
            (
                None
                if not present_630w2
                else
                type(
                    value_630w2
                ).__name__
            ),

        "redacted_value":
            (
                None
                if not present_630w2
                else
                _redact_prior_numeric_630w2(
                    value_630w2
                )
            ),
    }


# =============================================================================
# 2) BUILD DEDUPLICATED SOURCE CORPUS
# =============================================================================

source_corpus_630w2 = []
seen_sha_630w2 = set()


def _add_source_630w2(
    origin,
    sequence,
    text,
):
    if (
        not isinstance(
            text,
            str,
        )
        or
        len(text) < 100
    ):
        return

    sha = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if sha in seen_sha_630w2:
        return

    seen_sha_630w2.add(
        sha
    )

    source_corpus_630w2.append(
        {
            "origin":
                origin,

            "sequence":
                sequence,

            "sha256":
                sha,

            "text":
                text,
        }
    )


if "In" in globals():
    try:
        for idx, text in enumerate(In):
            _add_source_630w2(
                "In",
                idx,
                text,
            )
    except Exception:
        pass


for name, value in list(
    globals().items()
):
    if not isinstance(
        value,
        str,
    ):
        continue

    upper = name.upper()

    if any(
        token in upper
        for token in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "PROTOCOL",
            "CONTRACT",
            "NOTEBOOK",
        )
    ):
        _add_source_630w2(
            "global:" + name,
            None,
            value,
        )


# =============================================================================
# 3) FD003-SPECIFIC SELECTION-METHODOLOGY SOURCE RANKING
# =============================================================================

signals_630w2 = (
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
    "accepted_mae",
    "abstained_mae",
    "selective_mae",
    "selective_gain",
    "relative_reduction",
    "coverage",
    "accepted_mask",
    "abstain",
    "threshold_candidates",
    "candidate_threshold",
    "risk <=",
    "risk <",
)


def _score_source_630w2(text):
    lower = text.lower()

    score = 0

    if "fd003" in lower:
        score += 20

    for signal in signals_630w2:
        count = lower.count(
            signal.lower()
        )

        if count:
            score += (
                min(
                    count,
                    8,
                )
                *
                5
            )

    restore_markers = (
        "frozen_restart",
        "restart checkpoint",
        "restore_flags",
    )

    if any(
        marker in lower
        for marker
        in restore_markers
    ):
        score -= 15

    return score


# =============================================================================
# 4) EXTRACT SELECTION-RELEVANT ASSIGNMENTS / CONTEXTS
# =============================================================================

selection_name_tokens_630w2 = (
    "threshold",
    "coverage",
    "accept",
    "abstain",
    "selective",
    "gain",
    "reduction",
    "policy",
)

context_regex_630w2 = re.compile(
    r"FD003_OPERATING_POLICY_CONTRACT_629AU|"
    r"FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW|"
    r"accepted_mae|"
    r"abstained_mae|"
    r"selective_mae|"
    r"selective_gain|"
    r"relative_reduction|"
    r"accepted_mask|"
    r"abstain|"
    r"coverage|"
    r"threshold_candidates|"
    r"candidate_threshold|"
    r"risk\s*<=|"
    r"risk\s*<",
    flags=re.IGNORECASE,
)


def _target_names_630w2(target):
    if isinstance(
        target,
        ast.Name,
    ):
        return [
            target.id
        ]

    if isinstance(
        target,
        (
            ast.Tuple,
            ast.List,
        ),
    ):
        out = []

        for elt in target.elts:
            out.extend(
                _target_names_630w2(
                    elt
                )
            )

        return out

    return []


def _selection_assignments_630w2(
    text
):
    out = []

    try:
        tree = ast.parse(
            text
        )
    except Exception:
        return out

    for node in ast.walk(tree):

        if not isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            ),
        ):
            continue

        if isinstance(
            node,
            ast.Assign,
        ):
            targets = []

            for target in node.targets:
                targets.extend(
                    _target_names_630w2(
                        target
                    )
                )

        else:
            targets = (
                _target_names_630w2(
                    node.target
                )
            )

        for target in targets:
            lower_target = (
                target.lower()
            )

            if not any(
                token
                in lower_target
                for token
                in selection_name_tokens_630w2
            ):
                continue

            source = (
                ast.get_source_segment(
                    text,
                    node,
                )
            )

            out.append(
                {
                    "target":
                        target,

                    "line":
                        int(
                            getattr(
                                node,
                                "lineno",
                                -1,
                            )
                        ),

                    "source":
                        source,
                }
            )

    return out


def _contexts_630w2(text):
    lines = text.splitlines()

    matches = [
        i
        for i, line
        in enumerate(
            lines,
            start=1,
        )
        if context_regex_630w2.search(
            line
        )
    ]

    contexts = []

    for line_no in matches[:20]:

        start = max(
            1,
            line_no - 4,
        )

        end = min(
            len(lines),
            line_no + 6,
        )

        contexts.append(
            {
                "match_line":
                    line_no,

                "snippet":
                    tuple(
                        (
                            i,
                            lines[
                                i - 1
                            ],
                        )
                        for i
                        in range(
                            start,
                            end + 1,
                        )
                    ),
            }
        )

    return contexts


# =============================================================================
# 5) RANK TRUE METHODOLOGY SOURCES
# =============================================================================

ranked_sources_630w2 = []

for record in source_corpus_630w2:

    score = _score_source_630w2(
        record[
            "text"
        ]
    )

    if score <= 0:
        continue

    assignments = (
        _selection_assignments_630w2(
            record[
                "text"
            ]
        )
    )

    contexts = _contexts_630w2(
        record[
            "text"
        ]
    )

    score += (
        3
        *
        len(
            assignments
        )
    )

    score += (
        2
        *
        len(
            contexts
        )
    )

    ranked_sources_630w2.append(
        {
            "origin":
                record[
                    "origin"
                ],

            "sequence":
                record[
                    "sequence"
                ],

            "sha256":
                record[
                    "sha256"
                ],

            "length":
                len(
                    record[
                        "text"
                    ]
                ),

            "score":
                int(
                    score
                ),

            "assignments":
                tuple(
                    assignments
                ),

            "contexts":
                tuple(
                    contexts
                ),
        }
    )


ranked_sources_630w2 = sorted(
    ranked_sources_630w2,
    key=lambda row: (
        -row[
            "score"
        ],
        -len(
            row[
                "assignments"
            ]
        ),
        -len(
            row[
                "contexts"
            ]
        ),
        row[
            "origin"
        ],
        (
            -1
            if row[
                "sequence"
            ]
            is None
            else
            row[
                "sequence"
            ]
        ),
    ),
)

top_sources_630w2 = tuple(
    ranked_sources_630w2[
        :12
    ]
)


# =============================================================================
# 6) FREEZE AUDIT
# =============================================================================

audit_payload_630w2 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-2",

    "status":
        "PRIOR_DATASET_POLICY_METHODOLOGY_RECOVERY_STATIC_ONLY",

    "w1_audit_exact":
        True,

    "prior_policy_contract_presence":
        {
            name:
                resident_contract_audit_630w2[
                    name
                ][
                    "present"
                ]
            for name
            in contract_names_630w2
        },

    "source_corpus_count":
        len(
            source_corpus_630w2
        ),

    "ranked_source_count":
        len(
            ranked_sources_630w2
        ),

    "top_source_fingerprints":
        tuple(
            {
                "origin":
                    row[
                        "origin"
                    ],

                "sequence":
                    row[
                        "sequence"
                    ],

                "sha256":
                    row[
                        "sha256"
                    ],

                "score":
                    row[
                        "score"
                    ],

                "assignment_targets":
                    tuple(
                        item[
                            "target"
                        ]
                        for item
                        in row[
                            "assignments"
                        ]
                    ),

                "context_match_lines":
                    tuple(
                        item[
                            "match_line"
                        ]
                        for item
                        in row[
                            "contexts"
                        ]
                    ),
            }
            for row
            in top_sources_630w2
        ),

    "prior_dataset_numeric_values_adopted":
        False,

    "fd004_dev_outcomes_computed":
        False,

    "fd004_threshold_selected":
        False,

    "fd004_policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_630W2 = (
    audit_payload_630w2
)

FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630w2,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w2 = True


# =============================================================================
# 7) REPORT — ARRANGED OUTPUT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-2 — FD004 GOVERNING THRESHOLD/POLICY "
    "METHODOLOGY RECOVERY"
)

print("=" * 176)

print(
    "FD004 DEV outcome computation     :",
    False,
)

print(
    "FD004 threshold selected          :",
    False,
)

print(
    "FD004 policy selected             :",
    False,
)

print(
    "Prior numeric threshold adopted   :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print(
    "\n--- RESIDENT PRIOR-DATASET POLICY CONTRACTS "
    "(NUMERICS REDACTED) ---"
)

for name in contract_names_630w2:

    entry = (
        resident_contract_audit_630w2[
            name
        ]
    )

    print(
        f"\n{name}"
    )

    print(
        "  present                         :",
        entry[
            "present"
        ],
    )

    print(
        "  type                            :",
        entry[
            "type"
        ],
    )

    print(
        "  redacted contract               :",
        entry[
            "redacted_value"
        ],
    )


print(
    "\n--- TOP FD003 METHODOLOGY SOURCE CANDIDATES ---"
)

print(
    "Source corpus count                :",
    len(
        source_corpus_630w2
    ),
)

print(
    "Ranked FD003 source count          :",
    len(
        ranked_sources_630w2
    ),
)


for idx, row in enumerate(
    top_sources_630w2,
    start=1,
):

    print(
        f"\nSOURCE {idx}"
    )

    print(
        "  origin                          :",
        row[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        row[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        row[
            "sha256"
        ],
    )

    print(
        "  score                           :",
        row[
            "score"
        ],
    )

    print(
        "  selection assignment targets    :",
        tuple(
            item[
                "target"
            ]
            for item
            in row[
                "assignments"
            ][
                :20
            ]
        ),
    )

    if row[
        "assignments"
    ]:

        print(
            "  exact assignment evidence:"
        )

        for item in row[
            "assignments"
        ][
            :10
        ]:

            source = (
                item[
                    "source"
                ]
                or
                ""
            )

            source_lines = (
                source.splitlines()
            )

            if len(
                source_lines
            ) > 8:

                source = "\n".join(
                    source_lines[
                        :8
                    ]
                    +
                    [
                        "    ..."
                    ]
                )

            print(
                f"    line "
                f"{item['line']}: "
                f"{source}"
            )

    if row[
        "contexts"
    ]:

        print(
            "  compact selection contexts:"
        )

        for context in row[
            "contexts"
        ][
            :8
        ]:

            print(
                f"    [match line "
                f"{context['match_line']}]"
            )

            for (
                line_no,
                line,
            ) in context[
                "snippet"
            ]:

                print(
                    f"      "
                    f"{line_no:05d} | "
                    f"{line[:220]}"
                )


print(
    "\n--- AUDIT FINGERPRINT ---"
)

print(
    "Methodology recovery SHA-256      :",
    FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2,
)

print(
    "\nPASS — prior-dataset operating-policy methodology was inspected "
    "statically with prior numeric values redacted. No FD004 DEV outcome "
    "or threshold was computed."
)

print(
    "Next: bind only the recovered methodology structure to FD004; "
    "derive any FD004 numeric threshold solely from FD004 DEV."
)

In [ ]:
# STEP 6.30W-2D — DISPLAY RESIDENT METHODOLOGY-RECOVERY EVIDENCE ONLY
#
# NO DEV outcomes.
# NO threshold selection.
# NO policy selection.
# NO TEST / RUL access.
# NO scientific execution.

import json
import hashlib


required_630w2d = (
    "all_checks_passed_630w2",
    "FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_630W2",
    "FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2",
    "resident_contract_audit_630w2",
    "top_sources_630w2",
)

missing_630w2d = [
    name
    for name in required_630w2d
    if name not in globals()
]

if missing_630w2d:
    raise RuntimeError(
        "Missing resident W-2 object(s): "
        + ", ".join(missing_630w2d)
    )

if not bool(all_checks_passed_630w2):
    raise RuntimeError(
        "STEP 6.30W-2 is not marked passed."
    )


audit_sha_recomputed_630w2d = hashlib.sha256(
    json.dumps(
        FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_630W2,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

audit_sha_exact_630w2d = bool(
    audit_sha_recomputed_630w2d
    ==
    str(
        FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2
    )
)

if not audit_sha_exact_630w2d:
    raise RuntimeError(
        "Resident W-2 methodology audit changed."
    )


print("=" * 176)
print(
    "STEP 6.30W-2D — RESIDENT THRESHOLD/POLICY "
    "METHODOLOGY EVIDENCE DISPLAY"
)
print("=" * 176)

print("Scientific execution performed    :", False)
print("FD004 DEV outcomes computed       :", False)
print("FD004 threshold selected          :", False)
print("FD004 policy selected             :", False)
print("Prior numeric threshold adopted   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)
print("W-2 audit SHA exact               :", audit_sha_exact_630w2d)


print(
    "\n--- PRIOR POLICY CONTRACTS — NUMERIC VALUES REDACTED ---"
)

for name_630w2d, entry_630w2d in (
    resident_contract_audit_630w2.items()
):
    print(f"\n{name_630w2d}")
    print(
        "  present                         :",
        entry_630w2d["present"],
    )
    print(
        "  type                            :",
        entry_630w2d["type"],
    )
    print(
        "  redacted contract:"
    )
    print(
        json.dumps(
            entry_630w2d["redacted_value"],
            indent=2,
            sort_keys=True,
            default=str,
        )
    )


print(
    "\n--- TOP RECOVERED METHODOLOGY SOURCES ---"
)

print(
    "Source count shown                :",
    len(top_sources_630w2),
)

for idx_630w2d, row_630w2d in enumerate(
    top_sources_630w2,
    start=1,
):

    print(f"\nSOURCE {idx_630w2d}")

    print(
        "  origin                          :",
        row_630w2d["origin"],
    )

    print(
        "  sequence                        :",
        row_630w2d["sequence"],
    )

    print(
        "  SHA-256                         :",
        row_630w2d["sha256"],
    )

    print(
        "  score                           :",
        row_630w2d["score"],
    )

    print(
        "  assignment targets              :",
        tuple(
            item["target"]
            for item
            in row_630w2d["assignments"]
        ),
    )

    if row_630w2d["assignments"]:
        print("  exact assignment evidence:")

        for item_630w2d in row_630w2d[
            "assignments"
        ][:12]:

            print(
                f"\n    LINE {item_630w2d['line']}"
            )

            print(
                item_630w2d["source"]
            )

    if row_630w2d["contexts"]:
        print("  exact source contexts:")

        for context_630w2d in row_630w2d[
            "contexts"
        ][:12]:

            print(
                f"\n    CONTEXT AROUND LINE "
                f"{context_630w2d['match_line']}"
            )

            for (
                line_no_630w2d,
                line_630w2d,
            ) in context_630w2d["snippet"]:

                print(
                    f"      {line_no_630w2d:05d} | "
                    f"{line_630w2d}"
                )


print("\n--- AUDIT ---")

print(
    "Methodology recovery SHA-256      :",
    FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2,
)

print(
    "\nPASS — this cell only redisplayed the already-resident "
    "W-2 methodology evidence. Nothing scientific was executed."
)

In [ ]:
# STEP 6.30W-3 — ORIGINAL FD003 OPERATING-POLICY CONSTRUCTION SOURCE RECOVERY
#
# STATIC SOURCE AUDIT ONLY.
# NO FD004 DEV outcome arithmetic.
# NO FD004 threshold/policy selection.
# NO TEST / official RUL access.
#
# Purpose:
# Recover the ORIGINAL policy-construction source (not restart restoration)
# that created FD003_OPERATING_POLICY_CONTRACT_629AU, including:
# - nominal coverage / quantile level
# - quantile method
# - threshold source
# - acceptance comparator
# - tie handling
# - whether labels/errors/corruption evidence entered threshold selection.

import ast
import hashlib
import json
import re


required_630w3 = (
    "all_checks_passed_630w2",
    "FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_630W2",
    "FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2",
)

missing_630w3 = [
    name
    for name in required_630w3
    if name not in globals()
]

if missing_630w3:
    raise RuntimeError(
        "Missing STEP 6.30W-3 authority object(s): "
        + ", ".join(missing_630w3)
    )

if not bool(all_checks_passed_630w2):
    raise RuntimeError(
        "STEP 6.30W-2 must remain passed."
    )


w2_sha_630w3 = hashlib.sha256(
    json.dumps(
        FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_630W2,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    w2_sha_630w3
    !=
    str(
        FD004_POLICY_METHODOLOGY_RECOVERY_AUDIT_SHA256_630W2
    )
):
    raise RuntimeError(
        "STEP 6.30W-2 audit object changed."
    )


# =============================================================================
# 1) BUILD DEDUPLICATED STATIC SOURCE CORPUS
# =============================================================================

source_corpus_630w3 = []
seen_sha_630w3 = set()


def _add_source_630w3(
    origin,
    sequence,
    text,
):
    if (
        not isinstance(text, str)
        or
        len(text) < 100
    ):
        return

    sha = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if sha in seen_sha_630w3:
        return

    seen_sha_630w3.add(sha)

    source_corpus_630w3.append(
        {
            "origin":
                origin,

            "sequence":
                sequence,

            "sha256":
                sha,

            "text":
                text,
        }
    )


if "In" in globals():
    try:
        for idx, text in enumerate(In):
            _add_source_630w3(
                "In",
                idx,
                text,
            )
    except Exception:
        pass


for name, value in list(
    globals().items()
):
    if not isinstance(
        value,
        str,
    ):
        continue

    upper = name.upper()

    if any(
        token in upper
        for token in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "PROTOCOL",
            "CONTRACT",
            "NOTEBOOK",
        )
    ):
        _add_source_630w3(
            "global:" + name,
            None,
            value,
        )


# =============================================================================
# 2) FIND EXACT CONTRACT ASSIGNMENTS AND CLASSIFY RHS
# =============================================================================

TARGET_CONTRACT_630W3 = (
    "FD003_OPERATING_POLICY_CONTRACT_629AU"
)

assignment_hits_630w3 = []


def _assignment_target_names_630w3(
    node
):
    if isinstance(
        node,
        ast.Name,
    ):
        return [
            node.id
        ]

    if isinstance(
        node,
        (
            ast.Tuple,
            ast.List,
        ),
    ):
        names = []

        for elt in node.elts:
            names.extend(
                _assignment_target_names_630w3(
                    elt
                )
            )

        return names

    return []


for record in source_corpus_630w3:

    text = record[
        "text"
    ]

    if TARGET_CONTRACT_630W3 not in text:
        continue

    try:
        tree = ast.parse(
            text
        )
    except Exception:
        continue

    for node in ast.walk(
        tree
    ):

        if not isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            ),
        ):
            continue

        if isinstance(
            node,
            ast.Assign,
        ):
            names = []

            for target in node.targets:
                names.extend(
                    _assignment_target_names_630w3(
                        target
                    )
                )

            value_node = node.value

        else:
            names = (
                _assignment_target_names_630w3(
                    node.target
                )
            )

            value_node = node.value

        if (
            TARGET_CONTRACT_630W3
            not in names
        ):
            continue

        try:
            source_segment = (
                ast.get_source_segment(
                    text,
                    node,
                )
            )
        except Exception:
            source_segment = None

        rhs_type = (
            None
            if value_node is None
            else
            type(
                value_node
            ).__name__
        )

        assignment_hits_630w3.append(
            {
                "origin":
                    record[
                        "origin"
                    ],

                "sequence":
                    record[
                        "sequence"
                    ],

                "sha256":
                    record[
                        "sha256"
                    ],

                "text":
                    text,

                "line":
                    int(
                        getattr(
                            node,
                            "lineno",
                            -1,
                        )
                    ),

                "rhs_type":
                    rhs_type,

                "source":
                    source_segment,

                "is_dict_literal":
                    isinstance(
                        value_node,
                        ast.Dict,
                    ),

                "is_subscript_restore":
                    isinstance(
                        value_node,
                        ast.Subscript,
                    ),
            }
        )


# =============================================================================
# 3) LOCATE METHODOLOGY-SIGNAL LINES
# =============================================================================

signal_regex_630w3 = re.compile(
    r"nominal_coverage|"
    r"nominal_ceil_count|"
    r"quantile|"
    r"frozen_risk_threshold|"
    r"threshold_source|"
    r"acceptance_rule|"
    r"acceptance_comparator|"
    r"tie_handling|"
    r"dev_errors_used_for_threshold|"
    r"dev_labels_used_for_threshold|"
    r"corruption_.*used_for_threshold|"
    r"final_selective_risk|"
    r"accepted_mask|"
    r"abstain",
    flags=re.IGNORECASE,
)


def _extract_signal_contexts_630w3(
    text
):
    lines = text.splitlines()

    matching = [
        idx
        for idx, line
        in enumerate(
            lines,
            start=1,
        )
        if signal_regex_630w3.search(
            line
        )
    ]

    contexts = []

    for line_no in matching[
        :30
    ]:

        start = max(
            1,
            line_no - 4,
        )

        end = min(
            len(lines),
            line_no + 6,
        )

        contexts.append(
            {
                "match_line":
                    line_no,

                "snippet":
                    tuple(
                        (
                            idx,
                            lines[
                                idx - 1
                            ],
                        )
                        for idx
                        in range(
                            start,
                            end + 1,
                        )
                    ),
            }
        )

    return contexts


for hit in assignment_hits_630w3:

    hit[
        "contexts"
    ] = (
        _extract_signal_contexts_630w3(
            hit[
                "text"
            ]
        )
    )

    lower_text = (
        hit[
            "text"
        ].lower()
    )

    score = 0

    score += (
        100
        if hit[
            "is_dict_literal"
        ]
        else 0
    )

    score -= (
        50
        if hit[
            "is_subscript_restore"
        ]
        else 0
    )

    score += (
        15
        if "nominal_coverage"
        in lower_text
        else 0
    )

    score += (
        15
        if (
            "np.quantile"
            in lower_text
            or
            ".quantile("
            in lower_text
        )
        else 0
    )

    score += (
        10
        if "quantile_method"
        in lower_text
        else 0
    )

    score += (
        10
        if "threshold_source"
        in lower_text
        else 0
    )

    score += (
        10
        if "acceptance_rule"
        in lower_text
        else 0
    )

    score += (
        10
        if "tie_handling"
        in lower_text
        else 0
    )

    score += (
        10
        if "dev_errors_used_for_threshold"
        in lower_text
        else 0
    )

    if (
        "frozen_restart"
        in lower_text
        or
        "restore_flags"
        in lower_text
        or
        "hashes_restart"
        in lower_text
    ):
        score -= 40

    hit[
        "score"
    ] = int(
        score
    )


assignment_hits_630w3 = sorted(
    assignment_hits_630w3,
    key=lambda row: (
        -row[
            "score"
        ],
        (
            0
            if row[
                "is_dict_literal"
            ]
            else 1
        ),
        row[
            "origin"
        ],
        (
            -1
            if row[
                "sequence"
            ]
            is None
            else
            row[
                "sequence"
            ]
        ),
        row[
            "line"
        ],
    ),
)


# =============================================================================
# 4) FIND OTHER ORIGINAL-CONSTRUCTION CANDIDATES
# =============================================================================

construction_candidates_630w3 = []

for record in source_corpus_630w3:

    text = record[
        "text"
    ]

    lower = text.lower()

    if "fd003" not in lower:
        continue

    signal_count = sum(
        int(
            token in lower
        )
        for token in (
            "nominal_coverage",
            "np.quantile",
            "quantile_method",
            "threshold_source",
            "acceptance_rule",
            "tie_handling",
            "dev_errors_used_for_threshold",
            "frozen_risk_threshold",
        )
    )

    if signal_count < 4:
        continue

    restoration_penalty = int(
        (
            "frozen_restart"
            in lower
            or
            "hashes_restart"
            in lower
            or
            "restore_flags"
            in lower
        )
    )

    score = (
        10
        *
        signal_count
        -
        35
        *
        restoration_penalty
    )

    construction_candidates_630w3.append(
        {
            "origin":
                record[
                    "origin"
                ],

            "sequence":
                record[
                    "sequence"
                ],

            "sha256":
                record[
                    "sha256"
                ],

            "length":
                len(
                    text
                ),

            "score":
                int(
                    score
                ),

            "contexts":
                tuple(
                    _extract_signal_contexts_630w3(
                        text
                    )
                ),
        }
    )


construction_candidates_630w3 = sorted(
    construction_candidates_630w3,
    key=lambda row: (
        -row[
            "score"
        ],
        row[
            "origin"
        ],
        (
            -1
            if row[
                "sequence"
            ]
            is None
            else
            row[
                "sequence"
            ]
        ),
    ),
)


# =============================================================================
# 5) FREEZE AUDIT
# =============================================================================

audit_payload_630w3 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-3",

    "status":
        "ORIGINAL_FD003_POLICY_CONSTRUCTION_SOURCE_RECOVERY_STATIC_ONLY",

    "w2_audit_exact":
        True,

    "contract_assignment_hit_count":
        len(
            assignment_hits_630w3
        ),

    "dict_literal_assignment_count":
        sum(
            int(
                hit[
                    "is_dict_literal"
                ]
            )
            for hit
            in assignment_hits_630w3
        ),

    "subscript_restore_assignment_count":
        sum(
            int(
                hit[
                    "is_subscript_restore"
                ]
            )
            for hit
            in assignment_hits_630w3
        ),

    "construction_candidate_count":
        len(
            construction_candidates_630w3
        ),

    "top_assignment_fingerprints":
        tuple(
            {
                "origin":
                    hit[
                        "origin"
                    ],

                "sequence":
                    hit[
                        "sequence"
                    ],

                "sha256":
                    hit[
                        "sha256"
                    ],

                "line":
                    hit[
                        "line"
                    ],

                "rhs_type":
                    hit[
                        "rhs_type"
                    ],

                "is_dict_literal":
                    hit[
                        "is_dict_literal"
                    ],

                "is_subscript_restore":
                    hit[
                        "is_subscript_restore"
                    ],

                "score":
                    hit[
                        "score"
                    ],
            }
            for hit
            in assignment_hits_630w3[
                :10
            ]
        ),

    "prior_dataset_numeric_threshold_adopted":
        False,

    "fd004_dev_outcomes_computed":
        False,

    "fd004_threshold_selected":
        False,

    "fd004_policy_selected":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_630W3 = (
    audit_payload_630w3
)

FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_SHA256_630W3 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630w3,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w3 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-3 — ORIGINAL FD003 OPERATING-POLICY "
    "CONSTRUCTION SOURCE RECOVERY"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("FD004 DEV outcomes computed       :", False)
print("FD004 threshold selected          :", False)
print("FD004 policy selected             :", False)
print("Prior numeric threshold adopted   :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print(
    "\nContract assignment hits          :",
    len(
        assignment_hits_630w3
    ),
)

print(
    "Direct dict-literal constructions :",
    sum(
        int(
            hit[
                "is_dict_literal"
            ]
        )
        for hit
        in assignment_hits_630w3
    ),
)

print(
    "Restart/subscript restorations    :",
    sum(
        int(
            hit[
                "is_subscript_restore"
            ]
        )
        for hit
        in assignment_hits_630w3
    ),
)


print(
    "\n--- EXACT CONTRACT ASSIGNMENT HITS ---"
)

for idx, hit in enumerate(
    assignment_hits_630w3[
        :10
    ],
    start=1,
):

    print(
        f"\nASSIGNMENT {idx}"
    )

    print(
        "  origin                          :",
        hit[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        hit[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        hit[
            "sha256"
        ],
    )

    print(
        "  line                            :",
        hit[
            "line"
        ],
    )

    print(
        "  RHS AST type                    :",
        hit[
            "rhs_type"
        ],
    )

    print(
        "  direct dict construction        :",
        hit[
            "is_dict_literal"
        ],
    )

    print(
        "  restart/subscript restoration   :",
        hit[
            "is_subscript_restore"
        ],
    )

    print(
        "  score                           :",
        hit[
            "score"
        ],
    )

    print(
        "  exact assignment:"
    )

    print(
        hit[
            "source"
        ]
    )

    if hit[
        "contexts"
    ]:

        print(
            "  methodology contexts:"
        )

        for context in hit[
            "contexts"
        ][
            :12
        ]:

            print(
                f"\n    CONTEXT AROUND LINE "
                f"{context['match_line']}"
            )

            for (
                line_no,
                line,
            ) in context[
                "snippet"
            ]:

                print(
                    f"      "
                    f"{line_no:05d} | "
                    f"{line}"
                )


print(
    "\n--- OTHER ORIGINAL-CONSTRUCTION CANDIDATES ---"
)

print(
    "Candidate count                   :",
    len(
        construction_candidates_630w3
    ),
)

for idx, row in enumerate(
    construction_candidates_630w3[
        :8
    ],
    start=1,
):

    print(
        f"\nCANDIDATE {idx}"
    )

    print(
        "  origin                          :",
        row[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        row[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        row[
            "sha256"
        ],
    )

    print(
        "  score                           :",
        row[
            "score"
        ],
    )

    for context in row[
        "contexts"
    ][
        :10
    ]:

        print(
            f"\n    CONTEXT AROUND LINE "
            f"{context['match_line']}"
        )

        for (
            line_no,
            line,
        ) in context[
            "snippet"
        ]:

            print(
                f"      "
                f"{line_no:05d} | "
                f"{line}"
            )


print(
    "\n--- AUDIT ---"
)

print(
    "Original-construction audit SHA   :",
    FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_SHA256_630W3,
)

print(
    "\nPASS — only historical source was inspected. "
    "No FD004 outcome, threshold, policy, TEST, or official RUL was used."
)

print(
    "Next: bind the exact transferable threshold-selection methodology "
    "only after the original construction rule is visible here."
)

In [ ]:
# STEP 6.30W-4 — FD004 TRANSFERABLE POLICY-METHODOLOGY PARAMETER BIND
#
# CONTRACT READ ONLY.
# NO FD004 DEV outcome arithmetic.
# NO FD004 threshold selection.
# NO policy activation.
# NO TEST / official RUL access.
#
# Reads only methodology fields from the resident FD003 operating-policy
# contract. It deliberately does NOT read/print the prior frozen threshold
# or prior outcome metrics/counts.

import hashlib
import json

required_630w4 = (
    "all_checks_passed_630w3",
    "FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_630W3",
    "FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_SHA256_630W3",
    "FD003_OPERATING_POLICY_CONTRACT_629AU",
)

missing_630w4 = [
    name
    for name in required_630w4
    if name not in globals()
]

if missing_630w4:
    raise RuntimeError(
        "Missing STEP 6.30W-4 authority object(s): "
        + ", ".join(missing_630w4)
    )

if not bool(all_checks_passed_630w3):
    raise RuntimeError(
        "STEP 6.30W-3 must remain passed."
    )

w3_sha_recomputed_630w4 = hashlib.sha256(
    json.dumps(
        FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_630W3,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    w3_sha_recomputed_630w4
    !=
    str(
        FD004_ORIGINAL_POLICY_CONSTRUCTION_SOURCE_AUDIT_SHA256_630W3
    )
):
    raise RuntimeError(
        "STEP 6.30W-3 audit object changed."
    )

prior_contract_630w4 = FD003_OPERATING_POLICY_CONTRACT_629AU

if not isinstance(
    prior_contract_630w4,
    dict,
):
    raise RuntimeError(
        "Resident FD003 operating-policy contract is not a dict."
    )

methodology_keys_630w4 = (
    "policy_form",
    "nominal_coverage",
    "quantile_method",
    "quantile_input_dtype",
    "threshold_source",
    "risk_orientation",
    "acceptance_comparator",
    "acceptance_rule",
    "tie_handling",
    "dev_errors_used_for_threshold",
    "dev_labels_used_for_threshold",
    "corruption_predictions_used_for_threshold",
    "corruption_risks_used_for_threshold",
    "corruption_errors_used_for_threshold",
    "model_retrained",
    "risk_architecture_modified",
)

missing_methodology_keys_630w4 = [
    key
    for key in methodology_keys_630w4
    if key not in prior_contract_630w4
]

if missing_methodology_keys_630w4:
    raise RuntimeError(
        "Resident FD003 contract is missing methodology key(s): "
        + ", ".join(missing_methodology_keys_630w4)
    )

methodology_630w4 = {
    key:
        prior_contract_630w4[key]
    for key
    in methodology_keys_630w4
}

if (
    str(
        methodology_630w4[
            "policy_form"
        ]
    )
    !=
    "uncertainty_only_fixed_threshold"
):
    raise RuntimeError(
        "Unexpected policy form in resident prior contract."
    )

nominal_coverage_630w4 = float(
    methodology_630w4[
        "nominal_coverage"
    ]
)

if not (
    0.0
    <
    nominal_coverage_630w4
    <
    1.0
):
    raise RuntimeError(
        "Prior methodology nominal coverage is outside (0,1)."
    )

if (
    str(
        methodology_630w4[
            "quantile_method"
        ]
    )
    !=
    "linear"
):
    raise RuntimeError(
        "Unexpected quantile method."
    )

if (
    str(
        methodology_630w4[
            "acceptance_comparator"
        ]
    )
    !=
    "<="
):
    raise RuntimeError(
        "Unexpected acceptance comparator."
    )

if bool(
    methodology_630w4[
        "dev_errors_used_for_threshold"
    ]
):
    raise RuntimeError(
        "Prior methodology used DEV errors for threshold selection."
    )

if bool(
    methodology_630w4[
        "dev_labels_used_for_threshold"
    ]
):
    raise RuntimeError(
        "Prior methodology used DEV labels for threshold selection."
    )

for key_630w4 in (
    "corruption_predictions_used_for_threshold",
    "corruption_risks_used_for_threshold",
    "corruption_errors_used_for_threshold",
):
    if bool(
        methodology_630w4[
            key_630w4
        ]
    ):
        raise RuntimeError(
            f"Prior methodology unexpectedly used {key_630w4}."
        )


FD004_OPERATING_POLICY_METHODOLOGY_630W4 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-4",

    "policy_form":
        str(
            methodology_630w4[
                "policy_form"
            ]
        ),

    "nominal_coverage":
        nominal_coverage_630w4,

    "quantile_method":
        str(
            methodology_630w4[
                "quantile_method"
            ]
        ),

    "quantile_input_dtype":
        str(
            methodology_630w4[
                "quantile_input_dtype"
            ]
        ),

    "threshold_source":
        "clean FD004 DEV final selective risk only",

    "risk_orientation":
        str(
            methodology_630w4[
                "risk_orientation"
            ]
        ),

    "acceptance_comparator":
        str(
            methodology_630w4[
                "acceptance_comparator"
            ]
        ),

    "acceptance_rule":
        "accept iff FD004 final_selective_risk <= FD004 frozen_risk_threshold",

    "tie_handling":
        str(
            methodology_630w4[
                "tie_handling"
            ]
        ),

    "dev_errors_used_for_threshold":
        False,

    "dev_labels_used_for_threshold":
        False,

    "corruption_predictions_used_for_threshold":
        False,

    "corruption_risks_used_for_threshold":
        False,

    "corruption_errors_used_for_threshold":
        False,

    "prior_dataset_frozen_threshold_transferred":
        False,

    "prior_dataset_outcome_metrics_transferred":
        False,

    "fd004_threshold_selected":
        False,

    "fd004_policy_activated":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}

FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4 = (
    hashlib.sha256(
        json.dumps(
            FD004_OPERATING_POLICY_METHODOLOGY_630W4,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w4 = True


print("=" * 176)

print(
    "STEP 6.30W-4 — FD004 TRANSFERABLE "
    "POLICY-METHODOLOGY PARAMETER BIND"
)

print("=" * 176)

print("Scientific execution performed    :", False)
print("FD004 DEV outcomes computed       :", False)
print("FD004 threshold selected          :", False)
print("FD004 policy activated            :", False)
print("Prior frozen threshold transferred:", False)
print("Prior outcome metrics transferred :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- TRANSFERABLE METHODOLOGY ---")

print(
    "Policy form                       :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "policy_form"
    ],
)

print(
    "Nominal coverage / quantile level :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "nominal_coverage"
    ],
)

print(
    "Quantile method                   :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "quantile_method"
    ],
)

print(
    "Quantile input dtype              :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "quantile_input_dtype"
    ],
)

print(
    "Threshold source                  :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "threshold_source"
    ],
)

print(
    "Risk orientation                  :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "risk_orientation"
    ],
)

print(
    "Acceptance comparator             :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "acceptance_comparator"
    ],
)

print(
    "Tie handling                      :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "tie_handling"
    ],
)

print("\n--- THRESHOLD-SELECTION EXCLUSIONS ---")

print(
    "DEV labels used                   :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "dev_labels_used_for_threshold"
    ],
)

print(
    "DEV errors used                   :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "dev_errors_used_for_threshold"
    ],
)

print(
    "Corruption predictions used       :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "corruption_predictions_used_for_threshold"
    ],
)

print(
    "Corruption risks used             :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "corruption_risks_used_for_threshold"
    ],
)

print(
    "Corruption errors used            :",
    FD004_OPERATING_POLICY_METHODOLOGY_630W4[
        "corruption_errors_used_for_threshold"
    ],
)

print("\n--- EVIDENCE LOCK ---")

print(
    "FD004 methodology SHA-256         :",
    FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4,
)

print(
    "\nPASS — the transferable operating-policy methodology is now "
    "bound for FD004 without transferring the prior dataset's frozen "
    "threshold or outcome metrics."
)

print(
    "Next: derive the FD004 numeric risk threshold from CLEAN FD004 DEV "
    "final selective risk only, still without using DEV labels/errors."
)

In [ ]:
# STEP 6.30W-5 — FD004 CLEAN-DEV RISK THRESHOLD / POLICY FREEZE
#
# LABEL-FREE POLICY FREEZE ONLY.
# NO DEV labels / targets.
# NO DEV prediction errors.
# NO MAE / RMSE / outcome metrics.
# NO corruption predictions / risks / errors.
# NO TEST / official RUL access.
#
# Threshold source:
# CLEAN FD004 DEV final selective risk only.
#
# Rule:
# q = transferable nominal coverage from W-4
# threshold = np.quantile(clean_dev_final_risk, q, method="linear")
# accept iff risk <= threshold
# exact threshold ties are accepted.

import numpy as np
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w5 = (
    "all_checks_passed_630w4",
    "FD004_OPERATING_POLICY_METHODOLOGY_630W4",
    "FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
    "sha256_array_630v2",
    "all_checks_passed_630v18p",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P",
)

missing_630w5 = [
    name
    for name in required_630w5
    if name not in globals()
]

if missing_630w5:
    raise RuntimeError(
        "Missing STEP 6.30W-5 authority object(s): "
        + ", ".join(missing_630w5)
    )

if not bool(all_checks_passed_630w4):
    raise RuntimeError(
        "STEP 6.30W-4 must remain passed."
    )

if not bool(all_checks_passed_630v18p):
    raise RuntimeError(
        "STEP 6.30V-18P must remain passed."
    )


# =============================================================================
# 1) VERIFY FROZEN METHODOLOGY + 381-SCENARIO CLOSURE
# =============================================================================

methodology_recomputed_sha_630w5 = hashlib.sha256(
    json.dumps(
        FD004_OPERATING_POLICY_METHODOLOGY_630W4,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

methodology_exact_630w5 = bool(
    methodology_recomputed_sha_630w5
    ==
    str(
        FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4
    )
)

if not methodology_exact_630w5:
    raise RuntimeError(
        "Frozen W-4 operating-policy methodology changed."
    )


corruption_closure_recomputed_sha_630w5 = hashlib.sha256(
    json.dumps(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

corruption_closure_exact_630w5 = bool(
    corruption_closure_recomputed_sha_630w5
    ==
    str(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P
    )
)

if not corruption_closure_exact_630w5:
    raise RuntimeError(
        "Frozen 381-scenario corruption closure changed."
    )


# =============================================================================
# 2) BIND METHODOLOGY — DO NOT READ PRIOR DATASET THRESHOLD
# =============================================================================

methodology_630w5 = dict(
    FD004_OPERATING_POLICY_METHODOLOGY_630W4
)

if (
    str(
        methodology_630w5[
            "policy_form"
        ]
    )
    !=
    "uncertainty_only_fixed_threshold"
):
    raise RuntimeError(
        "Unexpected FD004 policy form."
    )

nominal_coverage_630w5 = float(
    methodology_630w5[
        "nominal_coverage"
    ]
)

if not (
    0.0
    <
    nominal_coverage_630w5
    <
    1.0
):
    raise RuntimeError(
        "Nominal coverage is outside (0,1)."
    )

quantile_method_630w5 = str(
    methodology_630w5[
        "quantile_method"
    ]
)

if quantile_method_630w5 != "linear":
    raise RuntimeError(
        "Frozen quantile method is not 'linear'."
    )

if (
    str(
        methodology_630w5[
            "quantile_input_dtype"
        ]
    )
    !=
    "float64"
):
    raise RuntimeError(
        "Frozen quantile input dtype is not float64."
    )

if (
    str(
        methodology_630w5[
            "acceptance_comparator"
        ]
    )
    !=
    "<="
):
    raise RuntimeError(
        "Frozen acceptance comparator is not <=."
    )

for forbidden_key_630w5 in (
    "dev_errors_used_for_threshold",
    "dev_labels_used_for_threshold",
    "corruption_predictions_used_for_threshold",
    "corruption_risks_used_for_threshold",
    "corruption_errors_used_for_threshold",
):
    if bool(
        methodology_630w5[
            forbidden_key_630w5
        ]
    ):
        raise RuntimeError(
            "Frozen methodology violates threshold-selection exclusion: "
            + forbidden_key_630w5
        )


# =============================================================================
# 3) CLEAN DEV FINAL SELECTIVE RISK ONLY
# =============================================================================

clean_risk_630w5 = np.asarray(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
    dtype=np.float64,
).reshape(-1)

if clean_risk_630w5.shape != (10824,):
    raise RuntimeError(
        "Clean DEV final selective-risk shape changed."
    )

if not np.isfinite(
    clean_risk_630w5
).all():
    raise RuntimeError(
        "Clean DEV final selective risk contains non-finite values."
    )

if np.any(
    (clean_risk_630w5 < 0.0)
    |
    (clean_risk_630w5 > 1.0)
):
    raise RuntimeError(
        "Clean DEV final selective risk escaped [0,1]."
    )

clean_risk_sha_630w5 = str(
    sha256_array_630v2(
        clean_risk_630w5
    )
)

matching_resident_hash_authorities_630w5 = tuple(
    sorted(
        name
        for name, value
        in globals().items()
        if isinstance(
            value,
            str,
        )
        and
        value
        ==
        clean_risk_sha_630w5
        and
        name
        !=
        "clean_risk_sha_630w5"
    )
)


# =============================================================================
# 4) DERIVE FD004 NUMERIC THRESHOLD — LABEL-FREE
# =============================================================================

frozen_risk_threshold_630w5 = float(
    np.quantile(
        clean_risk_630w5,
        nominal_coverage_630w5,
        method=quantile_method_630w5,
    )
)

if not np.isfinite(
    frozen_risk_threshold_630w5
):
    raise RuntimeError(
        "Derived FD004 risk threshold is non-finite."
    )

if not (
    0.0
    <=
    frozen_risk_threshold_630w5
    <=
    1.0
):
    raise RuntimeError(
        "Derived FD004 risk threshold escaped [0,1]."
    )


# =============================================================================
# 5) FREEZE CLEAN-DEV ACCEPTANCE MASK — STILL LABEL-FREE
# =============================================================================

clean_accept_mask_630w5 = (
    clean_risk_630w5
    <=
    frozen_risk_threshold_630w5
)

clean_abstain_mask_630w5 = (
    ~clean_accept_mask_630w5
)

accepted_count_630w5 = int(
    np.sum(
        clean_accept_mask_630w5
    )
)

abstained_count_630w5 = int(
    np.sum(
        clean_abstain_mask_630w5
    )
)

realized_coverage_630w5 = float(
    np.mean(
        clean_accept_mask_630w5
    )
)

threshold_tie_count_630w5 = int(
    np.sum(
        clean_risk_630w5
        ==
        frozen_risk_threshold_630w5
    )
)

if (
    accepted_count_630w5
    +
    abstained_count_630w5
    !=
    clean_risk_630w5.size
):
    raise RuntimeError(
        "Acceptance/abstention partition is inconsistent."
    )

if accepted_count_630w5 <= 0:
    raise RuntimeError(
        "Frozen policy accepted zero clean DEV windows."
    )

if abstained_count_630w5 <= 0:
    raise RuntimeError(
        "Frozen policy abstained on zero clean DEV windows."
    )


# =============================================================================
# 6) FREEZE AUTHORITATIVE FD004 POLICY OBJECTS
# =============================================================================

FD004_FROZEN_RISK_THRESHOLD_630W5 = (
    frozen_risk_threshold_630w5
)

FD004_CLEAN_DEV_ACCEPT_MASK_630W5 = (
    clean_accept_mask_630w5.copy()
)

FD004_CLEAN_DEV_ABSTAIN_MASK_630W5 = (
    clean_abstain_mask_630w5.copy()
)

FD004_CLEAN_DEV_RISK_SHA256_630W5 = (
    clean_risk_sha_630w5
)

FD004_CLEAN_DEV_ACCEPT_MASK_SHA256_630W5 = str(
    sha256_array_630v2(
        FD004_CLEAN_DEV_ACCEPT_MASK_630W5
    )
)


FD004_FROZEN_OPERATING_POLICY_630W5 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-5",

    "status":
        "FD004_OPERATING_POLICY_FROZEN_LABEL_FREE",

    "policy_form":
        str(
            methodology_630w5[
                "policy_form"
            ]
        ),

    "nominal_coverage":
        nominal_coverage_630w5,

    "quantile_method":
        quantile_method_630w5,

    "quantile_input_dtype":
        "float64",

    "threshold_source":
        "clean FD004 DEV final selective risk only",

    "clean_dev_risk_sha256":
        FD004_CLEAN_DEV_RISK_SHA256_630W5,

    "frozen_risk_threshold":
        FD004_FROZEN_RISK_THRESHOLD_630W5,

    "risk_orientation":
        str(
            methodology_630w5[
                "risk_orientation"
            ]
        ),

    "acceptance_comparator":
        "<=",

    "acceptance_rule":
        "accept iff final_selective_risk <= frozen_risk_threshold",

    "tie_handling":
        str(
            methodology_630w5[
                "tie_handling"
            ]
        ),

    "clean_dev_population":
        int(
            clean_risk_630w5.size
        ),

    "clean_dev_accepted_count":
        accepted_count_630w5,

    "clean_dev_abstained_count":
        abstained_count_630w5,

    "clean_dev_realized_coverage":
        realized_coverage_630w5,

    "clean_dev_threshold_tie_count":
        threshold_tie_count_630w5,

    "clean_dev_accept_mask_sha256":
        FD004_CLEAN_DEV_ACCEPT_MASK_SHA256_630W5,

    "dev_labels_used_for_threshold":
        False,

    "dev_errors_used_for_threshold":
        False,

    "corruption_predictions_used_for_threshold":
        False,

    "corruption_risks_used_for_threshold":
        False,

    "corruption_errors_used_for_threshold":
        False,

    "prior_dataset_frozen_threshold_transferred":
        False,

    "threshold_selected":
        True,

    "policy_selected":
        True,

    "policy_frozen_before_dev_outcome_analysis":
        True,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_FROZEN_OPERATING_POLICY_SHA256_630W5 = (
    hashlib.sha256(
        json.dumps(
            FD004_FROZEN_OPERATING_POLICY_630W5,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_THRESHOLD_SELECTED_630W5 = True
FD004_POLICY_SELECTED_630W5 = True
FD004_DEV_TARGETS_ERRORS_USED_630W5 = False
FD004_TEST_CONTENTS_READ_630W5 = False
FD004_RUL_CONTENTS_READ_630W5 = False

all_checks_passed_630w5 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-5 — FD004 CLEAN-DEV RISK "
    "THRESHOLD / POLICY FREEZE"
)

print("=" * 176)

print("DEV labels used                   :", False)
print("DEV errors used                   :", False)
print("Corruption evidence used          :", False)
print("Prior frozen threshold transferred:", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- CLEAN DEV RISK AUTHORITY ---")

print(
    "Population                        :",
    clean_risk_630w5.size,
)

print(
    "Shape                             :",
    clean_risk_630w5.shape,
)

print(
    "dtype used for quantile           :",
    clean_risk_630w5.dtype,
)

print(
    "Risk min / median / mean / max    :",
    (
        float(np.min(clean_risk_630w5)),
        float(np.median(clean_risk_630w5)),
        float(np.mean(clean_risk_630w5)),
        float(np.max(clean_risk_630w5)),
    ),
)

print(
    "Clean risk SHA-256                :",
    FD004_CLEAN_DEV_RISK_SHA256_630W5,
)

print(
    "Matching resident hash names      :",
    matching_resident_hash_authorities_630w5,
)


print("\n--- FROZEN POLICY ---")

print(
    "Nominal coverage / quantile       :",
    nominal_coverage_630w5,
)

print(
    "Quantile method                   :",
    quantile_method_630w5,
)

print(
    "Frozen risk threshold             :",
    FD004_FROZEN_RISK_THRESHOLD_630W5,
)

print(
    "Acceptance rule                   :",
    "risk <= threshold",
)

print(
    "Exact threshold ties              :",
    threshold_tie_count_630w5,
)

print(
    "Accepted clean DEV windows        :",
    accepted_count_630w5,
)

print(
    "Abstained clean DEV windows       :",
    abstained_count_630w5,
)

print(
    "Realized clean DEV coverage       :",
    realized_coverage_630w5,
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Accept-mask SHA-256               :",
    FD004_CLEAN_DEV_ACCEPT_MASK_SHA256_630W5,
)

print(
    "Frozen policy SHA-256             :",
    FD004_FROZEN_OPERATING_POLICY_SHA256_630W5,
)

print(
    "Threshold selected                :",
    FD004_THRESHOLD_SELECTED_630W5,
)

print(
    "Policy selected                   :",
    FD004_POLICY_SELECTED_630W5,
)

print(
    "DEV targets/errors used           :",
    FD004_DEV_TARGETS_ERRORS_USED_630W5,
)

print(
    "\nPASS — the FD004 uncertainty-only operating policy is now "
    "numerically frozen from CLEAN FD004 DEV final selective risk only."
)

print(
    "Next: with the threshold/policy frozen, DEV labels/errors may be "
    "used for the pre-specified outcome/reporting analysis. TEST/RUL "
    "remain sealed."
)

In [ ]:
# STEP 6.30W-6A — FD004 DEV OUTCOME INPUT ALIGNMENT / EVIDENCE BIND
#
# POLICY IS ALREADY FROZEN IN W-5.
#
# This cell MAY read the frozen DEV target vector now.
# It does NOT compute:
# - prediction error
# - absolute error
# - MAE / RMSE
# - selective gain
# - accepted/abstained outcome metrics
#
# NO threshold modification.
# NO policy modification.
# NO corruption outcome analysis.
# NO TEST / official RUL access.

import numpy as np
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w6a = (
    "all_checks_passed_630w5",
    "FD004_FROZEN_OPERATING_POLICY_630W5",
    "FD004_FROZEN_OPERATING_POLICY_SHA256_630W5",
    "FD004_FROZEN_RISK_THRESHOLD_630W5",
    "FD004_CLEAN_DEV_ACCEPT_MASK_630W5",
    "FD004_CLEAN_DEV_ABSTAIN_MASK_630W5",
    "FD004_CLEAN_DEV_RISK_SHA256_630W5",
    "FD004_CLEAN_DEV_ACCEPT_MASK_SHA256_630W5",
    "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",
    "FD004_CLEAN_DEV_PREDICTION_630K",
    "y_DEV_FD004_630G",
    "DEV_WINDOW_UNIT_IDS_FD004_630G",
    "DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G",
    "sha256_array_630v2",
)

missing_630w6a = [
    name
    for name in required_630w6a
    if name not in globals()
]

if missing_630w6a:
    raise RuntimeError(
        "Missing STEP 6.30W-6A authority object(s): "
        + ", ".join(missing_630w6a)
    )

if not bool(all_checks_passed_630w5):
    raise RuntimeError(
        "STEP 6.30W-5 must remain passed."
    )


# =============================================================================
# 1) VERIFY FROZEN W-5 POLICY OBJECT
# =============================================================================

policy_recomputed_sha_630w6a = hashlib.sha256(
    json.dumps(
        FD004_FROZEN_OPERATING_POLICY_630W5,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

policy_sha_exact_630w6a = bool(
    policy_recomputed_sha_630w6a
    ==
    str(
        FD004_FROZEN_OPERATING_POLICY_SHA256_630W5
    )
)

if not policy_sha_exact_630w6a:
    raise RuntimeError(
        "Frozen W-5 operating-policy object changed."
    )

if not bool(
    FD004_FROZEN_OPERATING_POLICY_630W5.get(
        "policy_frozen_before_dev_outcome_analysis"
    )
):
    raise RuntimeError(
        "Policy was not documented as frozen before DEV outcome analysis."
    )

if not bool(
    FD004_FROZEN_OPERATING_POLICY_630W5.get(
        "threshold_selected"
    )
):
    raise RuntimeError(
        "FD004 threshold is not marked selected."
    )

if not bool(
    FD004_FROZEN_OPERATING_POLICY_630W5.get(
        "policy_selected"
    )
):
    raise RuntimeError(
        "FD004 policy is not marked selected."
    )


# =============================================================================
# 2) BIND DEV OUTCOME INPUTS — NO ERROR ARITHMETIC
# =============================================================================

target_630w6a = np.asarray(
    y_DEV_FD004_630G,
    dtype=np.float64,
).reshape(-1)

prediction_630w6a = np.asarray(
    FD004_CLEAN_DEV_PREDICTION_630K,
    dtype=np.float64,
).reshape(-1)

risk_630w6a = np.asarray(
    FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R,
    dtype=np.float64,
).reshape(-1)

accept_mask_630w6a = np.asarray(
    FD004_CLEAN_DEV_ACCEPT_MASK_630W5,
    dtype=bool,
).reshape(-1)

abstain_mask_630w6a = np.asarray(
    FD004_CLEAN_DEV_ABSTAIN_MASK_630W5,
    dtype=bool,
).reshape(-1)

unit_ids_630w6a = np.asarray(
    DEV_WINDOW_UNIT_IDS_FD004_630G,
    dtype=np.int64,
).reshape(-1)

endpoint_cycles_630w6a = np.asarray(
    DEV_WINDOW_ENDPOINT_CYCLES_FD004_630G,
    dtype=np.int64,
).reshape(-1)


# =============================================================================
# 3) SHAPE / FINITE / PARTITION CHECKS
# =============================================================================

expected_shape_630w6a = (
    10824,
)

shape_checks_630w6a = {
    "target":
        target_630w6a.shape
        ==
        expected_shape_630w6a,

    "prediction":
        prediction_630w6a.shape
        ==
        expected_shape_630w6a,

    "risk":
        risk_630w6a.shape
        ==
        expected_shape_630w6a,

    "accept_mask":
        accept_mask_630w6a.shape
        ==
        expected_shape_630w6a,

    "abstain_mask":
        abstain_mask_630w6a.shape
        ==
        expected_shape_630w6a,

    "unit_ids":
        unit_ids_630w6a.shape
        ==
        expected_shape_630w6a,

    "endpoint_cycles":
        endpoint_cycles_630w6a.shape
        ==
        expected_shape_630w6a,
}

if not all(
    shape_checks_630w6a.values()
):
    failed_630w6a = [
        key
        for key, passed
        in shape_checks_630w6a.items()
        if not passed
    ]

    raise RuntimeError(
        "DEV outcome-input shape check failed: "
        + ", ".join(failed_630w6a)
    )


finite_checks_630w6a = {
    "target":
        bool(
            np.isfinite(
                target_630w6a
            ).all()
        ),

    "prediction":
        bool(
            np.isfinite(
                prediction_630w6a
            ).all()
        ),

    "risk":
        bool(
            np.isfinite(
                risk_630w6a
            ).all()
        ),
}

if not all(
    finite_checks_630w6a.values()
):
    raise RuntimeError(
        "DEV outcome input contains non-finite numeric values."
    )


partition_checks_630w6a = {
    "logical_complements":
        bool(
            np.array_equal(
                abstain_mask_630w6a,
                ~accept_mask_630w6a,
            )
        ),

    "accepted_count":
        int(
            np.sum(
                accept_mask_630w6a
            )
        )
        ==
        int(
            FD004_FROZEN_OPERATING_POLICY_630W5[
                "clean_dev_accepted_count"
            ]
        ),

    "abstained_count":
        int(
            np.sum(
                abstain_mask_630w6a
            )
        )
        ==
        int(
            FD004_FROZEN_OPERATING_POLICY_630W5[
                "clean_dev_abstained_count"
            ]
        ),
}

if not all(
    partition_checks_630w6a.values()
):
    raise RuntimeError(
        "Frozen clean DEV acceptance partition changed."
    )


# =============================================================================
# 4) REVERIFY RISK + ACCEPTANCE MASK AGAINST W-5
# =============================================================================

risk_sha_630w6a = str(
    sha256_array_630v2(
        risk_630w6a
    )
)

accept_mask_sha_630w6a = str(
    sha256_array_630v2(
        accept_mask_630w6a
    )
)

risk_sha_exact_630w6a = bool(
    risk_sha_630w6a
    ==
    str(
        FD004_CLEAN_DEV_RISK_SHA256_630W5
    )
)

accept_mask_sha_exact_630w6a = bool(
    accept_mask_sha_630w6a
    ==
    str(
        FD004_CLEAN_DEV_ACCEPT_MASK_SHA256_630W5
    )
)

if not risk_sha_exact_630w6a:
    raise RuntimeError(
        "Clean DEV final selective-risk vector changed after W-5."
    )

if not accept_mask_sha_exact_630w6a:
    raise RuntimeError(
        "Frozen clean DEV accept mask changed after W-5."
    )


threshold_reproduces_mask_630w6a = bool(
    np.array_equal(
        accept_mask_630w6a,
        risk_630w6a
        <=
        float(
            FD004_FROZEN_RISK_THRESHOLD_630W5
        ),
    )
)

if not threshold_reproduces_mask_630w6a:
    raise RuntimeError(
        "Frozen threshold no longer reproduces the frozen accept mask."
    )


# =============================================================================
# 5) HASH THE THREE OUTCOME INPUT AUTHORITIES
#
# Hashing is identity evidence only.
# No target-prediction arithmetic is performed.
# =============================================================================

target_sha_630w6a = str(
    sha256_array_630v2(
        target_630w6a
    )
)

prediction_sha_630w6a = str(
    sha256_array_630v2(
        prediction_630w6a
    )
)


def _matching_hash_names_630w6a(
    hash_value,
    excluded_names,
):
    return tuple(
        sorted(
            name
            for name, value
            in globals().items()
            if (
                isinstance(
                    value,
                    str,
                )
                and
                value
                ==
                hash_value
                and
                name
                not in excluded_names
            )
        )
    )


target_hash_names_630w6a = (
    _matching_hash_names_630w6a(
        target_sha_630w6a,
        {
            "target_sha_630w6a",
        },
    )
)

prediction_hash_names_630w6a = (
    _matching_hash_names_630w6a(
        prediction_sha_630w6a,
        {
            "prediction_sha_630w6a",
        },
    )
)

risk_hash_names_630w6a = (
    _matching_hash_names_630w6a(
        risk_sha_630w6a,
        {
            "risk_sha_630w6a",
            "FD004_CLEAN_DEV_RISK_SHA256_630W5",
        },
    )
)


# =============================================================================
# 6) FREEZE OUTCOME-INPUT BINDING
# =============================================================================

FD004_DEV_OUTCOME_TARGET_630W6A = (
    target_630w6a.copy()
)

FD004_DEV_OUTCOME_CLEAN_PREDICTION_630W6A = (
    prediction_630w6a.copy()
)

FD004_DEV_OUTCOME_CLEAN_RISK_630W6A = (
    risk_630w6a.copy()
)

FD004_DEV_OUTCOME_ACCEPT_MASK_630W6A = (
    accept_mask_630w6a.copy()
)

FD004_DEV_OUTCOME_ABSTAIN_MASK_630W6A = (
    abstain_mask_630w6a.copy()
)

FD004_DEV_OUTCOME_UNIT_IDS_630W6A = (
    unit_ids_630w6a.copy()
)

FD004_DEV_OUTCOME_ENDPOINT_CYCLES_630W6A = (
    endpoint_cycles_630w6a.copy()
)


FD004_DEV_OUTCOME_INPUT_BINDING_630W6A = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-6A",

    "status":
        "DEV_OUTCOME_INPUTS_BOUND_AFTER_POLICY_FREEZE",

    "population":
        int(
            target_630w6a.size
        ),

    "target_source":
        "y_DEV_FD004_630G",

    "prediction_source":
        "FD004_CLEAN_DEV_PREDICTION_630K",

    "risk_source":
        "FD004_CLEAN_DEV_FINAL_SELECTIVE_RISK_630R",

    "accept_mask_source":
        "FD004_CLEAN_DEV_ACCEPT_MASK_630W5",

    "target_sha256":
        target_sha_630w6a,

    "prediction_sha256":
        prediction_sha_630w6a,

    "risk_sha256":
        risk_sha_630w6a,

    "accept_mask_sha256":
        accept_mask_sha_630w6a,

    "frozen_policy_sha256":
        str(
            FD004_FROZEN_OPERATING_POLICY_SHA256_630W5
        ),

    "frozen_risk_threshold":
        float(
            FD004_FROZEN_RISK_THRESHOLD_630W5
        ),

    "accepted_count":
        int(
            np.sum(
                accept_mask_630w6a
            )
        ),

    "abstained_count":
        int(
            np.sum(
                abstain_mask_630w6a
            )
        ),

    "dev_labels_accessed_after_policy_freeze":
        True,

    "dev_errors_computed":
        False,

    "dev_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "corruption_outcomes_used":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_DEV_OUTCOME_INPUT_BINDING_SHA256_630W6A = (
    hashlib.sha256(
        json.dumps(
            FD004_DEV_OUTCOME_INPUT_BINDING_630W6A,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w6a = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-6A — FD004 DEV OUTCOME "
    "INPUT ALIGNMENT / EVIDENCE BIND"
)

print("=" * 176)

print(
    "Policy already frozen             :",
    True,
)

print(
    "DEV labels accessed               :",
    True,
)

print(
    "DEV errors computed               :",
    False,
)

print(
    "DEV outcome metrics computed      :",
    False,
)

print(
    "Threshold modified                :",
    False,
)

print(
    "Policy modified                   :",
    False,
)

print(
    "Corruption outcomes used          :",
    False,
)

print(
    "TEST contents read                :",
    False,
)

print(
    "Official RUL contents read        :",
    False,
)


print("\n--- INPUT AUTHORITIES ---")

print(
    "Population                        :",
    target_630w6a.size,
)

print(
    "Target shape / dtype              :",
    (
        target_630w6a.shape,
        target_630w6a.dtype,
    ),
)

print(
    "Prediction shape / dtype          :",
    (
        prediction_630w6a.shape,
        prediction_630w6a.dtype,
    ),
)

print(
    "Risk shape / dtype                :",
    (
        risk_630w6a.shape,
        risk_630w6a.dtype,
    ),
)

print(
    "Accepted / abstained              :",
    (
        int(
            np.sum(
                accept_mask_630w6a
            )
        ),
        int(
            np.sum(
                abstain_mask_630w6a
            )
        ),
    ),
)

print(
    "Frozen threshold                  :",
    FD004_FROZEN_RISK_THRESHOLD_630W5,
)


print("\n--- HASH EVIDENCE ---")

print(
    "Target SHA-256                    :",
    target_sha_630w6a,
)

print(
    "Matching target hash names        :",
    target_hash_names_630w6a,
)

print(
    "Prediction SHA-256                :",
    prediction_sha_630w6a,
)

print(
    "Matching prediction hash names    :",
    prediction_hash_names_630w6a,
)

print(
    "Risk SHA exact to W-5             :",
    risk_sha_exact_630w6a,
)

print(
    "Other matching risk hash names    :",
    risk_hash_names_630w6a,
)

print(
    "Accept-mask SHA exact to W-5      :",
    accept_mask_sha_exact_630w6a,
)

print(
    "Threshold reproduces accept mask  :",
    threshold_reproduces_mask_630w6a,
)


print("\n--- ALIGNMENT ---")

print(
    "All shapes exact                  :",
    all(
        shape_checks_630w6a.values()
    ),
)

print(
    "All numeric inputs finite         :",
    all(
        finite_checks_630w6a.values()
    ),
)

print(
    "Acceptance partition exact        :",
    all(
        partition_checks_630w6a.values()
    ),
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Outcome-input binding SHA-256     :",
    FD004_DEV_OUTCOME_INPUT_BINDING_SHA256_630W6A,
)

print(
    "\nPASS — FD004 DEV target, clean prediction, clean risk, "
    "and frozen acceptance partition are now bound in exact common "
    "window order after the policy freeze."
)

print(
    "No prediction error or DEV outcome metric was computed."
)

print(
    "Next: compute the pre-specified CLEAN DEV full/accepted/abstained "
    "MAE reporting metrics under this already-frozen policy."
)

In [ ]:
# STEP 6.30W-6B — FD004 CLEAN DEV SELECTIVE OUTCOME REPORTING
#
# POLICY ALREADY FROZEN IN W-5.
# INPUT ORDER / AUTHORITIES BOUND IN W-6A.
#
# This cell computes CLEAN DEV reporting outcomes only:
# - full MAE
# - accepted MAE
# - abstained MAE
# - selective MAE gain = full MAE - accepted MAE
# - relative MAE reduction = selective gain / full MAE
# - abstained-minus-accepted MAE
#
# NO threshold modification.
# NO policy modification.
# NO corruption outcome analysis.
# NO TEST / official RUL access.

import numpy as np
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w6b = (
    "all_checks_passed_630w6a",
    "FD004_DEV_OUTCOME_INPUT_BINDING_630W6A",
    "FD004_DEV_OUTCOME_INPUT_BINDING_SHA256_630W6A",
    "FD004_DEV_OUTCOME_TARGET_630W6A",
    "FD004_DEV_OUTCOME_CLEAN_PREDICTION_630W6A",
    "FD004_DEV_OUTCOME_CLEAN_RISK_630W6A",
    "FD004_DEV_OUTCOME_ACCEPT_MASK_630W6A",
    "FD004_DEV_OUTCOME_ABSTAIN_MASK_630W6A",
    "FD004_FROZEN_OPERATING_POLICY_630W5",
    "FD004_FROZEN_OPERATING_POLICY_SHA256_630W5",
    "FD004_FROZEN_RISK_THRESHOLD_630W5",
    "sha256_array_630v2",
)

missing_630w6b = [
    name
    for name in required_630w6b
    if name not in globals()
]

if missing_630w6b:
    raise RuntimeError(
        "Missing STEP 6.30W-6B authority object(s): "
        + ", ".join(missing_630w6b)
    )

if not bool(all_checks_passed_630w6a):
    raise RuntimeError(
        "STEP 6.30W-6A must remain passed."
    )


# =============================================================================
# 1) VERIFY W-6A BINDING + W-5 POLICY HAVE NOT CHANGED
# =============================================================================

binding_recomputed_sha_630w6b = hashlib.sha256(
    json.dumps(
        FD004_DEV_OUTCOME_INPUT_BINDING_630W6A,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

binding_exact_630w6b = bool(
    binding_recomputed_sha_630w6b
    ==
    str(
        FD004_DEV_OUTCOME_INPUT_BINDING_SHA256_630W6A
    )
)

if not binding_exact_630w6b:
    raise RuntimeError(
        "Frozen W-6A outcome-input binding changed."
    )


policy_recomputed_sha_630w6b = hashlib.sha256(
    json.dumps(
        FD004_FROZEN_OPERATING_POLICY_630W5,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

policy_exact_630w6b = bool(
    policy_recomputed_sha_630w6b
    ==
    str(
        FD004_FROZEN_OPERATING_POLICY_SHA256_630W5
    )
)

if not policy_exact_630w6b:
    raise RuntimeError(
        "Frozen W-5 operating policy changed."
    )


# =============================================================================
# 2) BIND FROZEN CLEAN DEV OUTCOME INPUTS
# =============================================================================

target_630w6b = np.asarray(
    FD004_DEV_OUTCOME_TARGET_630W6A,
    dtype=np.float64,
).reshape(-1)

prediction_630w6b = np.asarray(
    FD004_DEV_OUTCOME_CLEAN_PREDICTION_630W6A,
    dtype=np.float64,
).reshape(-1)

risk_630w6b = np.asarray(
    FD004_DEV_OUTCOME_CLEAN_RISK_630W6A,
    dtype=np.float64,
).reshape(-1)

accept_mask_630w6b = np.asarray(
    FD004_DEV_OUTCOME_ACCEPT_MASK_630W6A,
    dtype=bool,
).reshape(-1)

abstain_mask_630w6b = np.asarray(
    FD004_DEV_OUTCOME_ABSTAIN_MASK_630W6A,
    dtype=bool,
).reshape(-1)

if not (
    target_630w6b.shape
    ==
    prediction_630w6b.shape
    ==
    risk_630w6b.shape
    ==
    accept_mask_630w6b.shape
    ==
    abstain_mask_630w6b.shape
    ==
    (10824,)
):
    raise RuntimeError(
        "Frozen CLEAN DEV outcome inputs are no longer aligned."
    )

if not np.array_equal(
    accept_mask_630w6b,
    risk_630w6b
    <=
    float(
        FD004_FROZEN_RISK_THRESHOLD_630W5
    ),
):
    raise RuntimeError(
        "Frozen threshold no longer reproduces the bound accept mask."
    )

if not np.array_equal(
    abstain_mask_630w6b,
    ~accept_mask_630w6b,
):
    raise RuntimeError(
        "Accept/abstain masks are no longer exact complements."
    )


# =============================================================================
# 3) COMPUTE CLEAN DEV ERROR VECTOR
# =============================================================================

signed_error_630w6b = (
    prediction_630w6b
    -
    target_630w6b
)

absolute_error_630w6b = np.abs(
    signed_error_630w6b
)

if not (
    np.isfinite(
        signed_error_630w6b
    ).all()
    and
    np.isfinite(
        absolute_error_630w6b
    ).all()
):
    raise RuntimeError(
        "CLEAN DEV error vector contains non-finite values."
    )


accepted_count_630w6b = int(
    np.sum(
        accept_mask_630w6b
    )
)

abstained_count_630w6b = int(
    np.sum(
        abstain_mask_630w6b
    )
)

if (
    accepted_count_630w6b
    !=
    int(
        FD004_FROZEN_OPERATING_POLICY_630W5[
            "clean_dev_accepted_count"
        ]
    )
):
    raise RuntimeError(
        "Frozen accepted-count changed."
    )

if (
    abstained_count_630w6b
    !=
    int(
        FD004_FROZEN_OPERATING_POLICY_630W5[
            "clean_dev_abstained_count"
        ]
    )
):
    raise RuntimeError(
        "Frozen abstained-count changed."
    )


# =============================================================================
# 4) PRE-SPECIFIED CLEAN DEV REPORTING METRICS
# =============================================================================

full_mae_630w6b = float(
    np.mean(
        absolute_error_630w6b
    )
)

accepted_mae_630w6b = float(
    np.mean(
        absolute_error_630w6b[
            accept_mask_630w6b
        ]
    )
)

abstained_mae_630w6b = float(
    np.mean(
        absolute_error_630w6b[
            abstain_mask_630w6b
        ]
    )
)

selective_mae_gain_630w6b = float(
    full_mae_630w6b
    -
    accepted_mae_630w6b
)

if full_mae_630w6b > 0.0:
    relative_mae_reduction_630w6b = float(
        selective_mae_gain_630w6b
        /
        full_mae_630w6b
    )
else:
    relative_mae_reduction_630w6b = 0.0

relative_mae_reduction_percent_630w6b = float(
    100.0
    *
    relative_mae_reduction_630w6b
)

abstained_minus_accepted_mae_630w6b = float(
    abstained_mae_630w6b
    -
    accepted_mae_630w6b
)

realized_coverage_630w6b = float(
    accepted_count_630w6b
    /
    absolute_error_630w6b.size
)


# Weighted reconstruction identity:
# full MAE must equal the population-weighted accepted/abstained MAEs.
weighted_reconstruction_630w6b = float(
    (
        accepted_count_630w6b
        *
        accepted_mae_630w6b
        +
        abstained_count_630w6b
        *
        abstained_mae_630w6b
    )
    /
    absolute_error_630w6b.size
)

weighted_reconstruction_exact_630w6b = bool(
    np.isclose(
        weighted_reconstruction_630w6b,
        full_mae_630w6b,
        rtol=0.0,
        atol=1e-12,
    )
)

if not weighted_reconstruction_exact_630w6b:
    raise RuntimeError(
        "Accepted/abstained MAEs do not reconstruct full CLEAN DEV MAE."
    )


# =============================================================================
# 5) FREEZE ERROR EVIDENCE + CLEAN DEV REPORT
# =============================================================================

FD004_CLEAN_DEV_SIGNED_ERROR_630W6B = (
    signed_error_630w6b.copy()
)

FD004_CLEAN_DEV_ABSOLUTE_ERROR_630W6B = (
    absolute_error_630w6b.copy()
)

FD004_CLEAN_DEV_SIGNED_ERROR_SHA256_630W6B = str(
    sha256_array_630v2(
        FD004_CLEAN_DEV_SIGNED_ERROR_630W6B
    )
)

FD004_CLEAN_DEV_ABSOLUTE_ERROR_SHA256_630W6B = str(
    sha256_array_630v2(
        FD004_CLEAN_DEV_ABSOLUTE_ERROR_630W6B
    )
)


FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-6B",

    "status":
        "CLEAN_DEV_SELECTIVE_OUTCOME_REPORTED_UNDER_FROZEN_POLICY",

    "population":
        int(
            absolute_error_630w6b.size
        ),

    "frozen_risk_threshold":
        float(
            FD004_FROZEN_RISK_THRESHOLD_630W5
        ),

    "nominal_coverage":
        float(
            FD004_FROZEN_OPERATING_POLICY_630W5[
                "nominal_coverage"
            ]
        ),

    "realized_coverage":
        realized_coverage_630w6b,

    "accepted_count":
        accepted_count_630w6b,

    "abstained_count":
        abstained_count_630w6b,

    "full_mae":
        full_mae_630w6b,

    "accepted_mae":
        accepted_mae_630w6b,

    "abstained_mae":
        abstained_mae_630w6b,

    "selective_mae_gain":
        selective_mae_gain_630w6b,

    "relative_mae_reduction":
        relative_mae_reduction_630w6b,

    "relative_mae_reduction_percent":
        relative_mae_reduction_percent_630w6b,

    "abstained_minus_accepted_mae":
        abstained_minus_accepted_mae_630w6b,

    "signed_error_sha256":
        FD004_CLEAN_DEV_SIGNED_ERROR_SHA256_630W6B,

    "absolute_error_sha256":
        FD004_CLEAN_DEV_ABSOLUTE_ERROR_SHA256_630W6B,

    "weighted_mae_reconstruction_exact":
        weighted_reconstruction_exact_630w6b,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "threshold_selected_before_dev_outcomes":
        True,

    "dev_labels_used_for_threshold":
        False,

    "dev_errors_used_for_threshold":
        False,

    "corruption_outcomes_used":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B = (
    hashlib.sha256(
        json.dumps(
            FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_DEV_OUTCOMES_COMPUTED_630W6B = True
FD004_THRESHOLD_MODIFIED_630W6B = False
FD004_POLICY_MODIFIED_630W6B = False
FD004_CORRUPTION_OUTCOMES_USED_630W6B = False
FD004_TEST_CONTENTS_READ_630W6B = False
FD004_RUL_CONTENTS_READ_630W6B = False

all_checks_passed_630w6b = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-6B — FD004 CLEAN DEV "
    "SELECTIVE OUTCOME REPORTING"
)

print("=" * 176)

print("Threshold modified                :", False)
print("Policy modified                   :", False)
print("Threshold frozen before outcomes  :", True)
print("Corruption outcomes used          :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FROZEN POLICY ---")

print(
    "Risk threshold                    :",
    FD004_FROZEN_RISK_THRESHOLD_630W5,
)

print(
    "Nominal coverage                  :",
    FD004_FROZEN_OPERATING_POLICY_630W5[
        "nominal_coverage"
    ],
)

print(
    "Realized coverage                 :",
    realized_coverage_630w6b,
)

print(
    "Accepted / abstained              :",
    (
        accepted_count_630w6b,
        abstained_count_630w6b,
    ),
)


print("\n--- CLEAN DEV OUTCOMES ---")

print(
    "Full MAE                          :",
    full_mae_630w6b,
)

print(
    "Accepted MAE                      :",
    accepted_mae_630w6b,
)

print(
    "Abstained MAE                     :",
    abstained_mae_630w6b,
)

print(
    "Selective MAE gain                :",
    selective_mae_gain_630w6b,
)

print(
    "Relative MAE reduction            :",
    relative_mae_reduction_630w6b,
)

print(
    "Relative MAE reduction (%)        :",
    relative_mae_reduction_percent_630w6b,
)

print(
    "Abstained - accepted MAE          :",
    abstained_minus_accepted_mae_630w6b,
)

print(
    "Weighted MAE reconstruction exact :",
    weighted_reconstruction_exact_630w6b,
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Signed-error SHA-256              :",
    FD004_CLEAN_DEV_SIGNED_ERROR_SHA256_630W6B,
)

print(
    "Absolute-error SHA-256            :",
    FD004_CLEAN_DEV_ABSOLUTE_ERROR_SHA256_630W6B,
)

print(
    "Outcome-report SHA-256            :",
    FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B,
)


print(
    "\nPASS — CLEAN FD004 DEV selective outcomes were reported "
    "under the already-frozen label-free operating policy."
)

print(
    "No threshold or policy value was changed, and TEST/RUL remain sealed."
)

print(
    "Next: analyze the already-frozen 381 corruption scenarios "
    "against these DEV outcomes under the same fixed threshold, "
    "without any retuning."
)

In [ ]:
# STEP 6.30W-7A — FD004 CORRUPTION POLICY-RESPONSE INPUT BIND
#
# CLEAN DEV policy already frozen in W-5.
# CLEAN DEV outcomes already reported in W-6B.
#
# This cell:
# - verifies all 381 frozen corruption prediction/risk matrices
# - verifies family manifests and scenario counts
# - applies the SAME frozen threshold to corruption risk
# - freezes per-scenario accept masks / coverage
#
# This cell DOES NOT:
# - read/use DEV labels for corruption outcomes
# - compute corruption errors or MAE
# - modify/reselect the threshold
# - modify the policy
# - access TEST or official RUL

import numpy as np
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w7a = (
    "all_checks_passed_630w6b",
    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B",
    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B",
    "FD004_FROZEN_OPERATING_POLICY_630W5",
    "FD004_FROZEN_OPERATING_POLICY_SHA256_630W5",
    "FD004_FROZEN_RISK_THRESHOLD_630W5",

    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P",

    "FD004_BIAS_SCENARIO_MANIFEST_630V18E",
    "FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E",
    "FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E",

    "FD004_NOISE_SCENARIO_MANIFEST_630V18H",
    "FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H",
    "FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H",

    "FD004_DRIFT_SCENARIO_MANIFEST_630V18J",
    "FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J",
    "FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J",

    "FD004_DROPOUT_SCENARIO_MANIFEST_630V18L",
    "FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L",
    "FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L",

    "FD004_STUCK_SCENARIO_MANIFEST_630V18N",
    "FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N",
    "FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N",

    "sha256_array_630v2",
)

missing_630w7a = [
    name
    for name in required_630w7a
    if name not in globals()
]

if missing_630w7a:
    raise RuntimeError(
        "Missing STEP 6.30W-7A authority object(s): "
        + ", ".join(missing_630w7a)
    )

if not bool(all_checks_passed_630w6b):
    raise RuntimeError(
        "STEP 6.30W-6B must remain passed."
    )


# =============================================================================
# 1) VERIFY PRIOR EVIDENCE LOCKS
# =============================================================================

clean_report_sha_recomputed_630w7a = hashlib.sha256(
    json.dumps(
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    clean_report_sha_recomputed_630w7a
    !=
    str(
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B
    )
):
    raise RuntimeError(
        "Frozen W-6B CLEAN DEV outcome report changed."
    )


policy_sha_recomputed_630w7a = hashlib.sha256(
    json.dumps(
        FD004_FROZEN_OPERATING_POLICY_630W5,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    policy_sha_recomputed_630w7a
    !=
    str(
        FD004_FROZEN_OPERATING_POLICY_SHA256_630W5
    )
):
    raise RuntimeError(
        "Frozen W-5 operating policy changed."
    )


corruption_closure_sha_recomputed_630w7a = hashlib.sha256(
    json.dumps(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    corruption_closure_sha_recomputed_630w7a
    !=
    str(
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P
    )
):
    raise RuntimeError(
        "Frozen 381-scenario corruption closure changed."
    )


frozen_threshold_630w7a = float(
    FD004_FROZEN_RISK_THRESHOLD_630W5
)


# =============================================================================
# 2) FAMILY AUTHORITIES
# =============================================================================

family_specs_630w7a = {
    "bias": {
        "expected_count": 102,
        "manifest": FD004_BIAS_SCENARIO_MANIFEST_630V18E,
        "prediction": FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E,
        "risk": FD004_BIAS_FINAL_SELECTIVE_RISK_630V18E,
    },

    "noise": {
        "expected_count": 51,
        "manifest": FD004_NOISE_SCENARIO_MANIFEST_630V18H,
        "prediction": FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,
        "risk": FD004_NOISE_FINAL_SELECTIVE_RISK_630V18H,
    },

    "drift": {
        "expected_count": 102,
        "manifest": FD004_DRIFT_SCENARIO_MANIFEST_630V18J,
        "prediction": FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J,
        "risk": FD004_DRIFT_FINAL_SELECTIVE_RISK_630V18J,
    },

    "dropout": {
        "expected_count": 63,
        "manifest": FD004_DROPOUT_SCENARIO_MANIFEST_630V18L,
        "prediction": FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L,
        "risk": FD004_DROPOUT_FINAL_SELECTIVE_RISK_630V18L,
    },

    "stuck": {
        "expected_count": 63,
        "manifest": FD004_STUCK_SCENARIO_MANIFEST_630V18N,
        "prediction": FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N,
        "risk": FD004_STUCK_FINAL_SELECTIVE_RISK_630V18N,
    },
}


# =============================================================================
# 3) SCENARIO-ID EXTRACTOR
# =============================================================================

def _scenario_ids_630w7a(manifest):
    if hasattr(manifest, "columns"):
        if "scenario_id" not in manifest.columns:
            raise RuntimeError(
                "Scenario manifest does not contain scenario_id."
            )

        return tuple(
            str(x)
            for x in manifest["scenario_id"].tolist()
        )

    if isinstance(manifest, dict):
        if "scenario_id" not in manifest:
            raise RuntimeError(
                "Scenario manifest dict does not contain scenario_id."
            )

        values = manifest["scenario_id"]

        if np.isscalar(values):
            return (
                str(values),
            )

        return tuple(
            str(x)
            for x in values
        )

    raise RuntimeError(
        "Unsupported scenario-manifest type: "
        + type(manifest).__name__
    )


# =============================================================================
# 4) VALIDATE MATRICES AND APPLY FROZEN POLICY
# =============================================================================

FD004_CORRUPTION_ACCEPT_MASKS_630W7A = {}
FD004_CORRUPTION_SCENARIO_IDS_630W7A = {}

family_binding_630w7a = {}

all_scenario_ids_630w7a = []

total_scenarios_630w7a = 0


for family_630w7a, spec_630w7a in family_specs_630w7a.items():

    expected_count_630w7a = int(
        spec_630w7a[
            "expected_count"
        ]
    )

    scenario_ids_630w7a = _scenario_ids_630w7a(
        spec_630w7a[
            "manifest"
        ]
    )

    if (
        len(
            scenario_ids_630w7a
        )
        !=
        expected_count_630w7a
    ):
        raise RuntimeError(
            f"{family_630w7a}: manifest count changed."
        )

    if (
        len(
            set(
                scenario_ids_630w7a
            )
        )
        !=
        expected_count_630w7a
    ):
        raise RuntimeError(
            f"{family_630w7a}: duplicate scenario IDs detected."
        )


    prediction_raw_630w7a = np.asarray(
        spec_630w7a[
            "prediction"
        ]
    )

    risk_raw_630w7a = np.asarray(
        spec_630w7a[
            "risk"
        ]
    )


    expected_shape_630w7a = (
        expected_count_630w7a,
        10824,
    )

    if (
        prediction_raw_630w7a.shape
        !=
        expected_shape_630w7a
    ):
        raise RuntimeError(
            f"{family_630w7a}: prediction shape changed: "
            f"{prediction_raw_630w7a.shape}"
        )

    if (
        risk_raw_630w7a.shape
        !=
        expected_shape_630w7a
    ):
        raise RuntimeError(
            f"{family_630w7a}: risk shape changed: "
            f"{risk_raw_630w7a.shape}"
        )


    if not np.isfinite(
        prediction_raw_630w7a
    ).all():
        raise RuntimeError(
            f"{family_630w7a}: non-finite predictions detected."
        )

    if not np.isfinite(
        risk_raw_630w7a
    ).all():
        raise RuntimeError(
            f"{family_630w7a}: non-finite risks detected."
        )

    if np.any(
        (risk_raw_630w7a < 0.0)
        |
        (risk_raw_630w7a > 1.0)
    ):
        raise RuntimeError(
            f"{family_630w7a}: risk escaped [0,1]."
        )


    prediction_sha_630w7a = str(
        sha256_array_630v2(
            prediction_raw_630w7a
        )
    )

    risk_sha_630w7a = str(
        sha256_array_630v2(
            risk_raw_630w7a
        )
    )


    accept_mask_630w7a = (
        np.asarray(
            risk_raw_630w7a,
            dtype=np.float64,
        )
        <=
        frozen_threshold_630w7a
    )

    coverage_per_scenario_630w7a = np.mean(
        accept_mask_630w7a,
        axis=1,
        dtype=np.float64,
    )

    accepted_count_per_scenario_630w7a = np.sum(
        accept_mask_630w7a,
        axis=1,
        dtype=np.int64,
    )


    if (
        accept_mask_630w7a.shape
        !=
        expected_shape_630w7a
    ):
        raise RuntimeError(
            f"{family_630w7a}: accept-mask shape changed."
        )

    if not np.isfinite(
        coverage_per_scenario_630w7a
    ).all():
        raise RuntimeError(
            f"{family_630w7a}: non-finite coverage detected."
        )


    accept_mask_sha_630w7a = str(
        sha256_array_630v2(
            accept_mask_630w7a
        )
    )

    scenario_ids_sha_630w7a = hashlib.sha256(
        json.dumps(
            list(
                scenario_ids_630w7a
            ),
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()


    FD004_CORRUPTION_ACCEPT_MASKS_630W7A[
        family_630w7a
    ] = accept_mask_630w7a.copy()

    FD004_CORRUPTION_SCENARIO_IDS_630W7A[
        family_630w7a
    ] = scenario_ids_630w7a


    family_binding_630w7a[
        family_630w7a
    ] = {
        "scenario_count":
            expected_count_630w7a,

        "window_count_per_scenario":
            10824,

        "scenario_ids_sha256":
            scenario_ids_sha_630w7a,

        "prediction_sha256":
            prediction_sha_630w7a,

        "risk_sha256":
            risk_sha_630w7a,

        "accept_mask_sha256":
            accept_mask_sha_630w7a,

        "accepted_count_min":
            int(
                np.min(
                    accepted_count_per_scenario_630w7a
                )
            ),

        "accepted_count_max":
            int(
                np.max(
                    accepted_count_per_scenario_630w7a
                )
            ),

        "coverage_min":
            float(
                np.min(
                    coverage_per_scenario_630w7a
                )
            ),

        "coverage_median":
            float(
                np.median(
                    coverage_per_scenario_630w7a
                )
            ),

        "coverage_mean":
            float(
                np.mean(
                    coverage_per_scenario_630w7a
                )
            ),

        "coverage_max":
            float(
                np.max(
                    coverage_per_scenario_630w7a
                )
            ),
    }


    all_scenario_ids_630w7a.extend(
        scenario_ids_630w7a
    )

    total_scenarios_630w7a += (
        expected_count_630w7a
    )


# =============================================================================
# 5) MASTER 381-SCENARIO CHECKS
# =============================================================================

if total_scenarios_630w7a != 381:
    raise RuntimeError(
        "Total executable corruption scenario count is not 381."
    )

if (
    len(
        all_scenario_ids_630w7a
    )
    !=
    381
):
    raise RuntimeError(
        "Combined corruption scenario-ID count is not 381."
    )

if (
    len(
        set(
            all_scenario_ids_630w7a
        )
    )
    !=
    381
):
    raise RuntimeError(
        "Combined corruption scenario IDs are not unique."
    )


combined_scenario_ids_sha_630w7a = hashlib.sha256(
    json.dumps(
        all_scenario_ids_630w7a,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()


# =============================================================================
# 6) FREEZE CORRUPTION POLICY-RESPONSE BINDING
# =============================================================================

FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-7A",

    "status":
        "381_CORRUPTION_POLICY_RESPONSE_BOUND_NO_OUTCOMES",

    "frozen_risk_threshold":
        frozen_threshold_630w7a,

    "acceptance_rule":
        "final_selective_risk <= frozen_risk_threshold",

    "scenario_count":
        381,

    "window_count_per_scenario":
        10824,

    "family_counts":
        {
            family:
                int(
                    spec[
                        "expected_count"
                    ]
                )
            for family, spec
            in family_specs_630w7a.items()
        },

    "family_binding":
        family_binding_630w7a,

    "combined_scenario_ids_sha256":
        combined_scenario_ids_sha_630w7a,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "threshold_recomputed":
        False,

    "dev_labels_used_in_this_stage":
        False,

    "corruption_errors_computed":
        False,

    "corruption_outcome_metrics_computed":
        False,

    "corruption_risks_used_for_threshold":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_POLICY_RESPONSE_BINDING_SHA256_630W7A = (
    hashlib.sha256(
        json.dumps(
            FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_CORRUPTION_POLICY_APPLIED_630W7A = True
FD004_CORRUPTION_ERRORS_COMPUTED_630W7A = False
FD004_CORRUPTION_OUTCOMES_COMPUTED_630W7A = False
FD004_THRESHOLD_MODIFIED_630W7A = False
FD004_POLICY_MODIFIED_630W7A = False
FD004_TEST_CONTENTS_READ_630W7A = False
FD004_RUL_CONTENTS_READ_630W7A = False

all_checks_passed_630w7a = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-7A — FD004 CORRUPTION "
    "POLICY-RESPONSE INPUT BIND"
)

print("=" * 176)

print("Frozen threshold                  :", frozen_threshold_630w7a)
print("Threshold modified                :", False)
print("Policy modified                   :", False)
print("Threshold recomputed              :", False)
print("DEV labels used in this stage     :", False)
print("Corruption errors computed        :", False)
print("Corruption outcome metrics        :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)

print("\n--- FAMILY POLICY RESPONSE ---")

for family_630w7a in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    row_630w7a = family_binding_630w7a[
        family_630w7a
    ]

    print(f"\n{family_630w7a.upper()}")

    print(
        "  scenarios                       :",
        row_630w7a[
            "scenario_count"
        ],
    )

    print(
        "  matrix shape                    :",
        (
            row_630w7a[
                "scenario_count"
            ],
            row_630w7a[
                "window_count_per_scenario"
            ],
        ),
    )

    print(
        "  coverage min/median/mean/max    :",
        (
            row_630w7a[
                "coverage_min"
            ],
            row_630w7a[
                "coverage_median"
            ],
            row_630w7a[
                "coverage_mean"
            ],
            row_630w7a[
                "coverage_max"
            ],
        ),
    )

    print(
        "  accepted-count min/max          :",
        (
            row_630w7a[
                "accepted_count_min"
            ],
            row_630w7a[
                "accepted_count_max"
            ],
        ),
    )

    print(
        "  prediction SHA-256              :",
        row_630w7a[
            "prediction_sha256"
        ],
    )

    print(
        "  risk SHA-256                    :",
        row_630w7a[
            "risk_sha256"
        ],
    )

    print(
        "  accept-mask SHA-256             :",
        row_630w7a[
            "accept_mask_sha256"
        ],
    )


print("\n--- MASTER CHECK ---")

print(
    "Total executable scenarios        :",
    total_scenarios_630w7a,
)

print(
    "Unique combined scenario IDs      :",
    len(
        set(
            all_scenario_ids_630w7a
        )
    ),
)

print(
    "Combined scenario-ID SHA-256      :",
    combined_scenario_ids_sha_630w7a,
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Policy-response binding SHA-256   :",
    FD004_CORRUPTION_POLICY_RESPONSE_BINDING_SHA256_630W7A,
)

print(
    "\nPASS — all 381 frozen corruption scenarios are now bound "
    "to the already-frozen FD004 operating policy."
)

print(
    "Only risk-based acceptance/coverage was computed; "
    "no corruption prediction error or outcome metric was used."
)

print(
    "Next: pair these already-frozen scenario predictions/masks "
    "with the bound DEV targets and compute per-scenario "
    "full/accepted/abstained MAE under the unchanged threshold."
)

In [ ]:
# STEP 6.30W-7B — FD004 381-SCENARIO CORRUPTION OUTCOME REPORTING
#
# The operating policy was frozen in W-5.
# Corruption policy response / accept masks were frozen in W-7A.
#
# This cell now pairs the already-frozen corruption predictions and
# accept masks with the already-bound DEV target vector and computes:
# - per-scenario full MAE
# - per-scenario accepted MAE
# - per-scenario abstained MAE
# - selective MAE gain
# - relative MAE reduction
# - abstained-minus-accepted MAE
#
# NO threshold selection/recomputation/modification.
# NO policy modification.
# NO new corruption execution.
# NO new model inference / MC dropout.
# NO TEST / official RUL access.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w7b = (
    "all_checks_passed_630w7a",
    "FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A",
    "FD004_CORRUPTION_POLICY_RESPONSE_BINDING_SHA256_630W7A",
    "FD004_CORRUPTION_ACCEPT_MASKS_630W7A",
    "FD004_CORRUPTION_SCENARIO_IDS_630W7A",

    "FD004_DEV_OUTCOME_TARGET_630W6A",
    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B",
    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B",

    "FD004_FROZEN_RISK_THRESHOLD_630W5",

    "FD004_BIAS_SCENARIO_MANIFEST_630V18E",
    "FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E",

    "FD004_NOISE_SCENARIO_MANIFEST_630V18H",
    "FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H",

    "FD004_DRIFT_SCENARIO_MANIFEST_630V18J",
    "FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J",

    "FD004_DROPOUT_SCENARIO_MANIFEST_630V18L",
    "FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L",

    "FD004_STUCK_SCENARIO_MANIFEST_630V18N",
    "FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N",

    "sha256_array_630v2",
)

missing_630w7b = [
    name
    for name in required_630w7b
    if name not in globals()
]

if missing_630w7b:
    raise RuntimeError(
        "Missing STEP 6.30W-7B authority object(s): "
        + ", ".join(missing_630w7b)
    )

if not bool(all_checks_passed_630w7a):
    raise RuntimeError(
        "STEP 6.30W-7A must remain passed."
    )


# =============================================================================
# 1) VERIFY W-7A + W-6B EVIDENCE LOCKS
# =============================================================================

binding_sha_recomputed_630w7b = hashlib.sha256(
    json.dumps(
        FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    binding_sha_recomputed_630w7b
    !=
    str(
        FD004_CORRUPTION_POLICY_RESPONSE_BINDING_SHA256_630W7A
    )
):
    raise RuntimeError(
        "Frozen W-7A corruption policy-response binding changed."
    )


clean_report_sha_recomputed_630w7b = hashlib.sha256(
    json.dumps(
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    clean_report_sha_recomputed_630w7b
    !=
    str(
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B
    )
):
    raise RuntimeError(
        "Frozen W-6B CLEAN DEV outcome report changed."
    )


frozen_threshold_630w7b = float(
    FD004_FROZEN_RISK_THRESHOLD_630W5
)

if (
    frozen_threshold_630w7b
    !=
    float(
        FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A[
            "frozen_risk_threshold"
        ]
    )
):
    raise RuntimeError(
        "W-7A threshold no longer matches the frozen W-5 threshold."
    )


# =============================================================================
# 2) BIND DEV TARGET
# =============================================================================

target_630w7b = np.asarray(
    FD004_DEV_OUTCOME_TARGET_630W6A,
    dtype=np.float64,
).reshape(-1)

if target_630w7b.shape != (10824,):
    raise RuntimeError(
        "Bound DEV target shape changed."
    )

if not np.isfinite(
    target_630w7b
).all():
    raise RuntimeError(
        "Bound DEV target contains non-finite values."
    )


# =============================================================================
# 3) FAMILY AUTHORITIES
# =============================================================================

family_specs_630w7b = {
    "bias": {
        "expected_count": 102,
        "manifest": FD004_BIAS_SCENARIO_MANIFEST_630V18E,
        "prediction": FD004_BIAS_DETERMINISTIC_PREDICTIONS_630V18E,
    },

    "noise": {
        "expected_count": 51,
        "manifest": FD004_NOISE_SCENARIO_MANIFEST_630V18H,
        "prediction": FD004_NOISE_DETERMINISTIC_PREDICTIONS_630V18H,
    },

    "drift": {
        "expected_count": 102,
        "manifest": FD004_DRIFT_SCENARIO_MANIFEST_630V18J,
        "prediction": FD004_DRIFT_DETERMINISTIC_PREDICTIONS_630V18J,
    },

    "dropout": {
        "expected_count": 63,
        "manifest": FD004_DROPOUT_SCENARIO_MANIFEST_630V18L,
        "prediction": FD004_DROPOUT_DETERMINISTIC_PREDICTIONS_630V18L,
    },

    "stuck": {
        "expected_count": 63,
        "manifest": FD004_STUCK_SCENARIO_MANIFEST_630V18N,
        "prediction": FD004_STUCK_DETERMINISTIC_PREDICTIONS_630V18N,
    },
}


def _manifest_frame_630w7b(
    manifest_630w7b,
    expected_count_630w7b,
):
    if hasattr(
        manifest_630w7b,
        "columns",
    ):
        frame_630w7b = (
            manifest_630w7b
            .copy()
            .reset_index(drop=True)
        )

    elif isinstance(
        manifest_630w7b,
        dict,
    ):
        try:
            frame_630w7b = pd.DataFrame(
                manifest_630w7b
            )
        except ValueError:
            frame_630w7b = pd.DataFrame(
                [
                    manifest_630w7b
                ]
            )

        frame_630w7b = (
            frame_630w7b
            .reset_index(drop=True)
        )

    else:
        raise RuntimeError(
            "Unsupported scenario-manifest type: "
            + type(
                manifest_630w7b
            ).__name__
        )

    if len(
        frame_630w7b
    ) != int(
        expected_count_630w7b
    ):
        raise RuntimeError(
            "Scenario-manifest row count changed."
        )

    if "scenario_id" not in frame_630w7b.columns:
        raise RuntimeError(
            "Scenario manifest does not contain scenario_id."
        )

    return frame_630w7b


# =============================================================================
# 4) COMPUTE PER-SCENARIO OUTCOMES FAMILY-BY-FAMILY
# =============================================================================

outcome_frames_630w7b = []
metric_rows_630w7b = []
combined_scenario_ids_630w7b = []

family_summaries_630w7b = {}

metric_columns_630w7b = (
    "accepted_count",
    "abstained_count",
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_mae_gain",
    "relative_mae_reduction",
    "relative_mae_reduction_percent",
    "abstained_minus_accepted_mae",
)


for family_630w7b, spec_630w7b in family_specs_630w7b.items():

    expected_count_630w7b = int(
        spec_630w7b[
            "expected_count"
        ]
    )

    manifest_frame_630w7b = _manifest_frame_630w7b(
        spec_630w7b[
            "manifest"
        ],
        expected_count_630w7b,
    )

    scenario_ids_630w7b = tuple(
        str(x)
        for x in manifest_frame_630w7b[
            "scenario_id"
        ].tolist()
    )

    frozen_ids_630w7b = tuple(
        str(x)
        for x in FD004_CORRUPTION_SCENARIO_IDS_630W7A[
            family_630w7b
        ]
    )

    if (
        scenario_ids_630w7b
        !=
        frozen_ids_630w7b
    ):
        raise RuntimeError(
            f"{family_630w7b}: manifest scenario order changed after W-7A."
        )


    prediction_630w7b = np.asarray(
        spec_630w7b[
            "prediction"
        ],
        dtype=np.float64,
    )

    accept_mask_630w7b = np.asarray(
        FD004_CORRUPTION_ACCEPT_MASKS_630W7A[
            family_630w7b
        ],
        dtype=bool,
    )

    expected_shape_630w7b = (
        expected_count_630w7b,
        10824,
    )

    if (
        prediction_630w7b.shape
        !=
        expected_shape_630w7b
    ):
        raise RuntimeError(
            f"{family_630w7b}: prediction shape changed."
        )

    if (
        accept_mask_630w7b.shape
        !=
        expected_shape_630w7b
    ):
        raise RuntimeError(
            f"{family_630w7b}: accept-mask shape changed."
        )

    if not np.isfinite(
        prediction_630w7b
    ).all():
        raise RuntimeError(
            f"{family_630w7b}: non-finite prediction detected."
        )


    absolute_error_630w7b = np.abs(
        prediction_630w7b
        -
        target_630w7b[
            None,
            :
        ]
    )

    if not np.isfinite(
        absolute_error_630w7b
    ).all():
        raise RuntimeError(
            f"{family_630w7b}: non-finite absolute error detected."
        )


    accepted_count_630w7b = np.sum(
        accept_mask_630w7b,
        axis=1,
        dtype=np.int64,
    )

    abstained_count_630w7b = (
        10824
        -
        accepted_count_630w7b
    )

    if np.any(
        accepted_count_630w7b
        <=
        0
    ):
        raise RuntimeError(
            f"{family_630w7b}: a scenario accepted zero DEV windows."
        )

    if np.any(
        abstained_count_630w7b
        <=
        0
    ):
        raise RuntimeError(
            f"{family_630w7b}: a scenario abstained on zero DEV windows."
        )


    coverage_630w7b = (
        accepted_count_630w7b.astype(
            np.float64
        )
        /
        10824.0
    )


    full_mae_630w7b = np.mean(
        absolute_error_630w7b,
        axis=1,
        dtype=np.float64,
    )


    accepted_error_sum_630w7b = np.sum(
        np.where(
            accept_mask_630w7b,
            absolute_error_630w7b,
            0.0,
        ),
        axis=1,
        dtype=np.float64,
    )

    accepted_mae_630w7b = (
        accepted_error_sum_630w7b
        /
        accepted_count_630w7b
    )


    abstain_mask_630w7b = (
        ~accept_mask_630w7b
    )

    abstained_error_sum_630w7b = np.sum(
        np.where(
            abstain_mask_630w7b,
            absolute_error_630w7b,
            0.0,
        ),
        axis=1,
        dtype=np.float64,
    )

    abstained_mae_630w7b = (
        abstained_error_sum_630w7b
        /
        abstained_count_630w7b
    )


    selective_gain_630w7b = (
        full_mae_630w7b
        -
        accepted_mae_630w7b
    )


    relative_reduction_630w7b = np.divide(
        selective_gain_630w7b,
        full_mae_630w7b,
        out=np.zeros_like(
            selective_gain_630w7b,
            dtype=np.float64,
        ),
        where=(
            full_mae_630w7b
            >
            0.0
        ),
    )

    relative_reduction_percent_630w7b = (
        100.0
        *
        relative_reduction_630w7b
    )


    abstained_minus_accepted_630w7b = (
        abstained_mae_630w7b
        -
        accepted_mae_630w7b
    )


    reconstructed_full_mae_630w7b = (
        (
            accepted_count_630w7b
            *
            accepted_mae_630w7b
        )
        +
        (
            abstained_count_630w7b
            *
            abstained_mae_630w7b
        )
    ) / 10824.0

    reconstruction_error_630w7b = np.abs(
        reconstructed_full_mae_630w7b
        -
        full_mae_630w7b
    )

    max_reconstruction_error_630w7b = float(
        np.max(
            reconstruction_error_630w7b
        )
    )

    if not np.allclose(
        reconstructed_full_mae_630w7b,
        full_mae_630w7b,
        rtol=0.0,
        atol=1e-10,
    ):
        raise RuntimeError(
            f"{family_630w7b}: selective MAEs do not reconstruct full MAE."
        )


    metric_matrix_family_630w7b = np.column_stack(
        (
            accepted_count_630w7b.astype(
                np.float64
            ),
            abstained_count_630w7b.astype(
                np.float64
            ),
            coverage_630w7b,
            full_mae_630w7b,
            accepted_mae_630w7b,
            abstained_mae_630w7b,
            selective_gain_630w7b,
            relative_reduction_630w7b,
            relative_reduction_percent_630w7b,
            abstained_minus_accepted_630w7b,
        )
    )

    if (
        metric_matrix_family_630w7b.shape
        !=
        (
            expected_count_630w7b,
            len(
                metric_columns_630w7b
            ),
        )
    ):
        raise RuntimeError(
            f"{family_630w7b}: metric-matrix shape changed."
        )

    if not np.isfinite(
        metric_matrix_family_630w7b
    ).all():
        raise RuntimeError(
            f"{family_630w7b}: non-finite scenario metric detected."
        )


    for metric_idx_630w7b, metric_name_630w7b in enumerate(
        metric_columns_630w7b
    ):
        if (
            metric_name_630w7b
            in
            manifest_frame_630w7b.columns
        ):
            raise RuntimeError(
                f"{family_630w7b}: manifest already contains reserved "
                f"reporting column {metric_name_630w7b!r}."
            )

        manifest_frame_630w7b[
            metric_name_630w7b
        ] = metric_matrix_family_630w7b[
            :,
            metric_idx_630w7b
        ]


    if (
        "analysis_family"
        in
        manifest_frame_630w7b.columns
    ):
        raise RuntimeError(
            f"{family_630w7b}: manifest already contains analysis_family."
        )

    manifest_frame_630w7b[
        "analysis_family"
    ] = family_630w7b


    outcome_frames_630w7b.append(
        manifest_frame_630w7b
    )

    metric_rows_630w7b.append(
        metric_matrix_family_630w7b
    )

    combined_scenario_ids_630w7b.extend(
        scenario_ids_630w7b
    )


    zero_gain_mask_630w7b = np.isclose(
        selective_gain_630w7b,
        0.0,
        rtol=0.0,
        atol=1e-12,
    )

    positive_gain_mask_630w7b = (
        selective_gain_630w7b
        >
        1e-12
    )

    negative_gain_mask_630w7b = (
        selective_gain_630w7b
        <
        -1e-12
    )


    if not np.all(
        positive_gain_mask_630w7b
        |
        zero_gain_mask_630w7b
        |
        negative_gain_mask_630w7b
    ):
        raise RuntimeError(
            f"{family_630w7b}: gain sign partition failed."
        )


    family_summaries_630w7b[
        family_630w7b
    ] = {
        "scenario_count":
            expected_count_630w7b,

        "coverage_min":
            float(
                np.min(
                    coverage_630w7b
                )
            ),

        "coverage_median":
            float(
                np.median(
                    coverage_630w7b
                )
            ),

        "coverage_mean":
            float(
                np.mean(
                    coverage_630w7b
                )
            ),

        "coverage_max":
            float(
                np.max(
                    coverage_630w7b
                )
            ),

        "full_mae_min":
            float(
                np.min(
                    full_mae_630w7b
                )
            ),

        "full_mae_median":
            float(
                np.median(
                    full_mae_630w7b
                )
            ),

        "full_mae_mean":
            float(
                np.mean(
                    full_mae_630w7b
                )
            ),

        "full_mae_max":
            float(
                np.max(
                    full_mae_630w7b
                )
            ),

        "accepted_mae_min":
            float(
                np.min(
                    accepted_mae_630w7b
                )
            ),

        "accepted_mae_median":
            float(
                np.median(
                    accepted_mae_630w7b
                )
            ),

        "accepted_mae_mean":
            float(
                np.mean(
                    accepted_mae_630w7b
                )
            ),

        "accepted_mae_max":
            float(
                np.max(
                    accepted_mae_630w7b
                )
            ),

        "selective_gain_min":
            float(
                np.min(
                    selective_gain_630w7b
                )
            ),

        "selective_gain_median":
            float(
                np.median(
                    selective_gain_630w7b
                )
            ),

        "selective_gain_mean":
            float(
                np.mean(
                    selective_gain_630w7b
                )
            ),

        "selective_gain_max":
            float(
                np.max(
                    selective_gain_630w7b
                )
            ),

        "relative_reduction_percent_mean":
            float(
                np.mean(
                    relative_reduction_percent_630w7b
                )
            ),

        "positive_gain_count":
            int(
                np.sum(
                    positive_gain_mask_630w7b
                )
            ),

        "zero_gain_count":
            int(
                np.sum(
                    zero_gain_mask_630w7b
                )
            ),

        "negative_gain_count":
            int(
                np.sum(
                    negative_gain_mask_630w7b
                )
            ),

        "max_weighted_reconstruction_error":
            max_reconstruction_error_630w7b,
    }


    del absolute_error_630w7b


# =============================================================================
# 5) MASTER 381-SCENARIO TABLE + HASH EVIDENCE
# =============================================================================

FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B = (
    pd.concat(
        outcome_frames_630w7b,
        axis=0,
        ignore_index=True,
        sort=False,
    )
)

FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B = (
    np.vstack(
        metric_rows_630w7b
    ).astype(
        np.float64,
        copy=False,
    )
)

FD004_CORRUPTION_METRIC_COLUMNS_630W7B = (
    metric_columns_630w7b
)


if len(
    FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B
) != 381:
    raise RuntimeError(
        "Combined corruption outcome table does not contain 381 rows."
    )

if (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B.shape
    !=
    (
        381,
        len(
            metric_columns_630w7b
        ),
    )
):
    raise RuntimeError(
        "Combined corruption metric matrix has unexpected shape."
    )


combined_scenario_ids_630w7b = tuple(
    combined_scenario_ids_630w7b
)

combined_scenario_ids_sha_630w7b = hashlib.sha256(
    json.dumps(
        list(
            combined_scenario_ids_630w7b
        ),
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

if (
    combined_scenario_ids_sha_630w7b
    !=
    str(
        FD004_CORRUPTION_POLICY_RESPONSE_BINDING_630W7A[
            "combined_scenario_ids_sha256"
        ]
    )
):
    raise RuntimeError(
        "Combined scenario order changed relative to W-7A."
    )


FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_SHA256_630W7B = str(
    sha256_array_630v2(
        FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B
    )
)


# =============================================================================
# 6) OVERALL REPORTING SUMMARY
# =============================================================================

metric_index_630w7b = {
    name:
        idx
    for idx, name
    in enumerate(
        metric_columns_630w7b
    )
}

coverage_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "coverage"
        ]
    ]
)

full_mae_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "full_mae"
        ]
    ]
)

accepted_mae_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "accepted_mae"
        ]
    ]
)

abstained_mae_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "abstained_mae"
        ]
    ]
)

gain_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "selective_mae_gain"
        ]
    ]
)

relative_percent_all_630w7b = (
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B[
        :,
        metric_index_630w7b[
            "relative_mae_reduction_percent"
        ]
    ]
)


zero_gain_all_630w7b = np.isclose(
    gain_all_630w7b,
    0.0,
    rtol=0.0,
    atol=1e-12,
)

positive_gain_all_630w7b = (
    gain_all_630w7b
    >
    1e-12
)

negative_gain_all_630w7b = (
    gain_all_630w7b
    <
    -1e-12
)


overall_summary_630w7b = {
    "scenario_count":
        381,

    "coverage_min":
        float(
            np.min(
                coverage_all_630w7b
            )
        ),

    "coverage_median":
        float(
            np.median(
                coverage_all_630w7b
            )
        ),

    "coverage_mean":
        float(
            np.mean(
                coverage_all_630w7b
            )
        ),

    "coverage_max":
        float(
            np.max(
                coverage_all_630w7b
            )
        ),

    "full_mae_min":
        float(
            np.min(
                full_mae_all_630w7b
            )
        ),

    "full_mae_median":
        float(
            np.median(
                full_mae_all_630w7b
            )
        ),

    "full_mae_mean":
        float(
            np.mean(
                full_mae_all_630w7b
            )
        ),

    "full_mae_max":
        float(
            np.max(
                full_mae_all_630w7b
            )
        ),

    "accepted_mae_min":
        float(
            np.min(
                accepted_mae_all_630w7b
            )
        ),

    "accepted_mae_median":
        float(
            np.median(
                accepted_mae_all_630w7b
            )
        ),

    "accepted_mae_mean":
        float(
            np.mean(
                accepted_mae_all_630w7b
            )
        ),

    "accepted_mae_max":
        float(
            np.max(
                accepted_mae_all_630w7b
            )
        ),

    "abstained_mae_mean":
        float(
            np.mean(
                abstained_mae_all_630w7b
            )
        ),

    "selective_gain_min":
        float(
            np.min(
                gain_all_630w7b
            )
        ),

    "selective_gain_median":
        float(
            np.median(
                gain_all_630w7b
            )
        ),

    "selective_gain_mean":
        float(
            np.mean(
                gain_all_630w7b
            )
        ),

    "selective_gain_max":
        float(
            np.max(
                gain_all_630w7b
            )
        ),

    "relative_reduction_percent_mean":
        float(
            np.mean(
                relative_percent_all_630w7b
            )
        ),

    "positive_gain_count":
        int(
            np.sum(
                positive_gain_all_630w7b
            )
        ),

    "zero_gain_count":
        int(
            np.sum(
                zero_gain_all_630w7b
            )
        ),

    "negative_gain_count":
        int(
            np.sum(
                negative_gain_all_630w7b
            )
        ),

    "positive_gain_rate":
        float(
            np.mean(
                positive_gain_all_630w7b
            )
        ),
}


if (
    overall_summary_630w7b[
        "positive_gain_count"
    ]
    +
    overall_summary_630w7b[
        "zero_gain_count"
    ]
    +
    overall_summary_630w7b[
        "negative_gain_count"
    ]
    !=
    381
):
    raise RuntimeError(
        "Overall selective-gain sign counts do not sum to 381."
    )


# =============================================================================
# 7) FREEZE REPORTING CONTRACT
# =============================================================================

FD004_CORRUPTION_OUTCOME_REPORT_630W7B = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-7B",

    "status":
        "381_CORRUPTION_OUTCOMES_REPORTED_UNDER_FROZEN_POLICY",

    "scenario_count":
        381,

    "window_count_per_scenario":
        10824,

    "frozen_risk_threshold":
        frozen_threshold_630w7b,

    "combined_scenario_ids_sha256":
        combined_scenario_ids_sha_630w7b,

    "metric_columns":
        metric_columns_630w7b,

    "metric_matrix_sha256":
        FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_SHA256_630W7B,

    "clean_dev_reference_full_mae":
        float(
            FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B[
                "full_mae"
            ]
        ),

    "clean_dev_reference_accepted_mae":
        float(
            FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B[
                "accepted_mae"
            ]
        ),

    "overall_summary":
        overall_summary_630w7b,

    "family_summaries":
        family_summaries_630w7b,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "corruption_outcomes_used_for_threshold":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B = (
    hashlib.sha256(
        json.dumps(
            FD004_CORRUPTION_OUTCOME_REPORT_630W7B,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)


FD004_CORRUPTION_OUTCOMES_COMPUTED_630W7B = True
FD004_THRESHOLD_MODIFIED_630W7B = False
FD004_POLICY_MODIFIED_630W7B = False
FD004_TEST_CONTENTS_READ_630W7B = False
FD004_RUL_CONTENTS_READ_630W7B = False

all_checks_passed_630w7b = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-7B — FD004 381-SCENARIO "
    "CORRUPTION OUTCOME REPORTING"
)

print("=" * 176)

print("Threshold modified                :", False)
print("Threshold recomputed              :", False)
print("Policy modified                   :", False)
print("New corruption executed           :", False)
print("New model inference               :", False)
print("New MC execution                  :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- MASTER OUTCOME SUMMARY ---")

print(
    "Scenario count                    :",
    overall_summary_630w7b[
        "scenario_count"
    ],
)

print(
    "Coverage min/median/mean/max      :",
    (
        overall_summary_630w7b[
            "coverage_min"
        ],
        overall_summary_630w7b[
            "coverage_median"
        ],
        overall_summary_630w7b[
            "coverage_mean"
        ],
        overall_summary_630w7b[
            "coverage_max"
        ],
    ),
)

print(
    "Full MAE min/median/mean/max      :",
    (
        overall_summary_630w7b[
            "full_mae_min"
        ],
        overall_summary_630w7b[
            "full_mae_median"
        ],
        overall_summary_630w7b[
            "full_mae_mean"
        ],
        overall_summary_630w7b[
            "full_mae_max"
        ],
    ),
)

print(
    "Accepted MAE min/median/mean/max  :",
    (
        overall_summary_630w7b[
            "accepted_mae_min"
        ],
        overall_summary_630w7b[
            "accepted_mae_median"
        ],
        overall_summary_630w7b[
            "accepted_mae_mean"
        ],
        overall_summary_630w7b[
            "accepted_mae_max"
        ],
    ),
)

print(
    "Selective gain min/median/mean/max:",
    (
        overall_summary_630w7b[
            "selective_gain_min"
        ],
        overall_summary_630w7b[
            "selective_gain_median"
        ],
        overall_summary_630w7b[
            "selective_gain_mean"
        ],
        overall_summary_630w7b[
            "selective_gain_max"
        ],
    ),
)

print(
    "Mean relative MAE reduction (%)   :",
    overall_summary_630w7b[
        "relative_reduction_percent_mean"
    ],
)

print(
    "Positive / zero / negative gain   :",
    (
        overall_summary_630w7b[
            "positive_gain_count"
        ],
        overall_summary_630w7b[
            "zero_gain_count"
        ],
        overall_summary_630w7b[
            "negative_gain_count"
        ],
    ),
)

print(
    "Positive-gain scenario rate       :",
    overall_summary_630w7b[
        "positive_gain_rate"
    ],
)


print("\n--- FAMILY SUMMARY ---")

for family_630w7b in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    row_630w7b = family_summaries_630w7b[
        family_630w7b
    ]

    print(f"\n{family_630w7b.upper()}")

    print(
        "  scenarios                       :",
        row_630w7b[
            "scenario_count"
        ],
    )

    print(
        "  full MAE median / mean          :",
        (
            row_630w7b[
                "full_mae_median"
            ],
            row_630w7b[
                "full_mae_mean"
            ],
        ),
    )

    print(
        "  accepted MAE median / mean      :",
        (
            row_630w7b[
                "accepted_mae_median"
            ],
            row_630w7b[
                "accepted_mae_mean"
            ],
        ),
    )

    print(
        "  gain min/median/mean/max        :",
        (
            row_630w7b[
                "selective_gain_min"
            ],
            row_630w7b[
                "selective_gain_median"
            ],
            row_630w7b[
                "selective_gain_mean"
            ],
            row_630w7b[
                "selective_gain_max"
            ],
        ),
    )

    print(
        "  positive / zero / negative      :",
        (
            row_630w7b[
                "positive_gain_count"
            ],
            row_630w7b[
                "zero_gain_count"
            ],
            row_630w7b[
                "negative_gain_count"
            ],
        ),
    )

    print(
        "  mean relative reduction (%)     :",
        row_630w7b[
            "relative_reduction_percent_mean"
        ],
    )

    print(
        "  max reconstruction error        :",
        row_630w7b[
            "max_weighted_reconstruction_error"
        ],
    )


print("\n--- EVIDENCE LOCK ---")

print(
    "Scenario metric matrix SHA-256    :",
    FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_SHA256_630W7B,
)

print(
    "Outcome report SHA-256            :",
    FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B,
)

print(
    "\nPASS — all 381 frozen corruption scenarios were evaluated "
    "under the unchanged FD004 operating policy."
)

print(
    "No threshold/policy retuning and no new corruption/model/MC "
    "execution occurred. TEST/RUL remain sealed."
)

print(
    "Next: summarize the frozen 381-scenario outcomes by family, "
    "severity, and sensor using this resident scenario-level report only."
)

In [ ]:
# STEP 6.30W-7C — FD004 CORRUPTION OUTCOME TABLE SCHEMA / GROUPING-FIELD AUDIT
#
# Purpose:
# Identify the exact resident manifest columns available for:
# - corruption family
# - sensor
# - severity
# - corruption parameters
#
# NO new scientific execution.
# NO new prediction/error computation.
# NO threshold/policy modification.
# NO TEST / official RUL access.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w7c = (
    "all_checks_passed_630w7b",
    "FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B",
    "FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_630W7B",
    "FD004_CORRUPTION_SCENARIO_METRIC_MATRIX_SHA256_630W7B",
    "FD004_CORRUPTION_OUTCOME_REPORT_630W7B",
    "FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B",
)

missing_630w7c = [
    name
    for name in required_630w7c
    if name not in globals()
]

if missing_630w7c:
    raise RuntimeError(
        "Missing STEP 6.30W-7C authority object(s): "
        + ", ".join(missing_630w7c)
    )

if not bool(all_checks_passed_630w7b):
    raise RuntimeError(
        "STEP 6.30W-7B must remain passed."
    )


# =============================================================================
# 1) VERIFY W-7B EVIDENCE LOCK
# =============================================================================

outcome_report_sha_recomputed_630w7c = hashlib.sha256(
    json.dumps(
        FD004_CORRUPTION_OUTCOME_REPORT_630W7B,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    outcome_report_sha_recomputed_630w7c
    !=
    str(
        FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B
    )
):
    raise RuntimeError(
        "Frozen W-7B corruption outcome report changed."
    )


# =============================================================================
# 2) BIND RESIDENT 381-ROW TABLE
# =============================================================================

table_630w7c = (
    FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B
    .copy()
    .reset_index(drop=True)
)

if len(table_630w7c) != 381:
    raise RuntimeError(
        "Resident corruption outcome table no longer has 381 rows."
    )

if "scenario_id" not in table_630w7c.columns:
    raise RuntimeError(
        "Resident corruption outcome table lost scenario_id."
    )

if "analysis_family" not in table_630w7c.columns:
    raise RuntimeError(
        "Resident corruption outcome table lost analysis_family."
    )

if table_630w7c["scenario_id"].astype(str).nunique() != 381:
    raise RuntimeError(
        "Scenario IDs are no longer unique."
    )


# =============================================================================
# 3) COLUMN AUDIT
# =============================================================================

metric_columns_630w7c = set(
    FD004_CORRUPTION_OUTCOME_REPORT_630W7B[
        "metric_columns"
    ]
)

known_reporting_columns_630w7c = (
    metric_columns_630w7c
    |
    {
        "analysis_family",
    }
)

manifest_columns_630w7c = tuple(
    col
    for col in table_630w7c.columns
    if col not in known_reporting_columns_630w7c
)

all_columns_630w7c = tuple(
    table_630w7c.columns.tolist()
)


# =============================================================================
# 4) IDENTIFY LIKELY GROUPING FIELDS BY NAME ONLY
# =============================================================================

def _name_matches_630w7c(
    column_name,
    tokens,
):
    lower = str(
        column_name
    ).lower()

    return any(
        token in lower
        for token in tokens
    )


sensor_candidate_columns_630w7c = tuple(
    col
    for col in manifest_columns_630w7c
    if _name_matches_630w7c(
        col,
        (
            "sensor",
            "feature",
            "channel",
        ),
    )
)

severity_candidate_columns_630w7c = tuple(
    col
    for col in manifest_columns_630w7c
    if _name_matches_630w7c(
        col,
        (
            "severity",
            "level",
            "magnitude",
            "sigma",
            "bias",
            "drift",
            "drop",
            "stuck",
            "value",
            "scale",
            "fraction",
            "rate",
        ),
    )
)

family_candidate_columns_630w7c = tuple(
    col
    for col in manifest_columns_630w7c
    if _name_matches_630w7c(
        col,
        (
            "family",
            "type",
            "corruption",
            "fault",
        ),
    )
)


# =============================================================================
# 5) COMPACT VALUE PROFILE FOR MANIFEST COLUMNS
# =============================================================================

column_profiles_630w7c = {}

for col_630w7c in manifest_columns_630w7c:

    series_630w7c = table_630w7c[
        col_630w7c
    ]

    nonnull_630w7c = series_630w7c.dropna()

    unique_count_630w7c = int(
        nonnull_630w7c.nunique(
            dropna=True
        )
    )

    sample_values_630w7c = tuple(
        str(x)
        for x in nonnull_630w7c.drop_duplicates().head(12).tolist()
    )

    column_profiles_630w7c[
        col_630w7c
    ] = {
        "dtype":
            str(
                series_630w7c.dtype
            ),

        "nonnull_count":
            int(
                nonnull_630w7c.shape[0]
            ),

        "unique_count":
            unique_count_630w7c,

        "sample_values":
            sample_values_630w7c,
    }


# =============================================================================
# 6) FAMILY-SPECIFIC MANIFEST PROFILE
# =============================================================================

family_manifest_profiles_630w7c = {}

for family_630w7c in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):

    subset_630w7c = table_630w7c.loc[
        table_630w7c[
            "analysis_family"
        ].astype(str)
        ==
        family_630w7c
    ].copy()

    family_manifest_profiles_630w7c[
        family_630w7c
    ] = {
        "rows":
            int(
                len(
                    subset_630w7c
                )
            ),

        "manifest_column_nonnull_counts":
            {
                col:
                    int(
                        subset_630w7c[
                            col
                        ].notna().sum()
                    )
                for col in manifest_columns_630w7c
            },
    }


# =============================================================================
# 7) FREEZE SCHEMA AUDIT
# =============================================================================

FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_630W7C = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-7C",

    "status":
        "CORRUPTION_OUTCOME_GROUPING_SCHEMA_AUDITED",

    "row_count":
        381,

    "all_columns":
        all_columns_630w7c,

    "manifest_columns":
        manifest_columns_630w7c,

    "sensor_candidate_columns":
        sensor_candidate_columns_630w7c,

    "severity_candidate_columns":
        severity_candidate_columns_630w7c,

    "family_candidate_columns":
        family_candidate_columns_630w7c,

    "column_profiles":
        column_profiles_630w7c,

    "family_manifest_profiles":
        family_manifest_profiles_630w7c,

    "new_outcome_computation":
        False,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_SHA256_630W7C = (
    hashlib.sha256(
        json.dumps(
            FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_630W7C,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w7c = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-7C — FD004 CORRUPTION OUTCOME "
    "TABLE SCHEMA / GROUPING-FIELD AUDIT"
)

print("=" * 176)

print("New outcome computation           :", False)
print("Threshold modified                :", False)
print("Policy modified                   :", False)
print("New corruption executed           :", False)
print("New model inference               :", False)
print("New MC execution                  :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- TABLE STRUCTURE ---")

print(
    "Rows                              :",
    len(
        table_630w7c
    ),
)

print(
    "Column count                      :",
    len(
        all_columns_630w7c
    ),
)

print(
    "All columns:"
)

for idx_630w7c, col_630w7c in enumerate(
    all_columns_630w7c,
    start=1,
):
    print(
        f"  {idx_630w7c:02d}. {col_630w7c}"
    )


print("\n--- MANIFEST / DESIGN COLUMNS ---")

print(
    "Manifest-column count             :",
    len(
        manifest_columns_630w7c
    ),
)

for col_630w7c in manifest_columns_630w7c:

    profile_630w7c = column_profiles_630w7c[
        col_630w7c
    ]

    print(
        f"\n{col_630w7c}"
    )

    print(
        "  dtype                           :",
        profile_630w7c[
            "dtype"
        ],
    )

    print(
        "  non-null / unique               :",
        (
            profile_630w7c[
                "nonnull_count"
            ],
            profile_630w7c[
                "unique_count"
            ],
        ),
    )

    print(
        "  sample unique values            :",
        profile_630w7c[
            "sample_values"
        ],
    )


print("\n--- GROUPING-FIELD CANDIDATES ---")

print(
    "Family candidates                 :",
    family_candidate_columns_630w7c,
)

print(
    "Sensor candidates                 :",
    sensor_candidate_columns_630w7c,
)

print(
    "Severity candidates               :",
    severity_candidate_columns_630w7c,
)


print("\n--- FAMILY ROW COUNTS ---")

for family_630w7c in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    print(
        f"{family_630w7c:<10}:",
        family_manifest_profiles_630w7c[
            family_630w7c
        ][
            "rows"
        ],
    )


print("\n--- EVIDENCE LOCK ---")

print(
    "Schema-audit SHA-256              :",
    FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_SHA256_630W7C,
)

print(
    "\nPASS — the resident 381-scenario outcome-table schema was "
    "audited without recomputing any scientific outcome."
)

print(
    "Next: use the exact discovered manifest fields to build "
    "family × severity and family × sensor summaries, including "
    "the 16 negative-gain stuck scenarios."
)

In [ ]:
# STEP 6.30W-7D — FD004 CORRUPTION FAMILY × SEVERITY / SENSOR SUMMARY
#
# Uses ONLY the frozen resident 381-row scenario outcome table from W-7B.
#
# Produces:
# 1) family × severity summary
# 2) family × sensor summary
# 3) exact 16 negative-gain STUCK scenarios
#
# NO new predictions.
# NO new errors from raw predictions.
# NO corruption re-execution.
# NO threshold/policy modification.
# NO TEST / official RUL access.

import numpy as np
import pandas as pd
import hashlib
import json


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w7d = (
    "all_checks_passed_630w7c",
    "FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_630W7C",
    "FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_SHA256_630W7C",
    "FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B",
    "FD004_CORRUPTION_OUTCOME_REPORT_630W7B",
    "FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B",
)

missing_630w7d = [
    name
    for name in required_630w7d
    if name not in globals()
]

if missing_630w7d:
    raise RuntimeError(
        "Missing STEP 6.30W-7D authority object(s): "
        + ", ".join(missing_630w7d)
    )

if not bool(all_checks_passed_630w7c):
    raise RuntimeError(
        "STEP 6.30W-7C must remain passed."
    )


# =============================================================================
# 1) VERIFY PRIOR LOCKS
# =============================================================================

schema_sha_recomputed_630w7d = hashlib.sha256(
    json.dumps(
        FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_630W7C,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    schema_sha_recomputed_630w7d
    !=
    str(
        FD004_CORRUPTION_GROUPING_SCHEMA_AUDIT_SHA256_630W7C
    )
):
    raise RuntimeError(
        "Frozen W-7C schema audit changed."
    )


outcome_sha_recomputed_630w7d = hashlib.sha256(
    json.dumps(
        FD004_CORRUPTION_OUTCOME_REPORT_630W7B,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    outcome_sha_recomputed_630w7d
    !=
    str(
        FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B
    )
):
    raise RuntimeError(
        "Frozen W-7B corruption outcome report changed."
    )


# =============================================================================
# 2) BIND RESIDENT SCENARIO TABLE
# =============================================================================

table_630w7d = (
    FD004_CORRUPTION_SCENARIO_OUTCOMES_630W7B
    .copy()
    .reset_index(drop=True)
)

required_columns_630w7d = (
    "scenario_id",
    "scenario_number",
    "family",
    "sensor",
    "severity",
    "alpha",
    "direction",
    "requested_fraction",
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_mae_gain",
    "relative_mae_reduction_percent",
    "abstained_minus_accepted_mae",
    "analysis_family",
)

missing_columns_630w7d = [
    col
    for col in required_columns_630w7d
    if col not in table_630w7d.columns
]

if missing_columns_630w7d:
    raise RuntimeError(
        "Missing required resident outcome-table column(s): "
        + ", ".join(missing_columns_630w7d)
    )

if len(table_630w7d) != 381:
    raise RuntimeError(
        "Resident scenario table no longer has 381 rows."
    )

if (
    table_630w7d["scenario_id"]
    .astype(str)
    .nunique()
    !=
    381
):
    raise RuntimeError(
        "Scenario IDs are no longer unique."
    )

if not np.array_equal(
    table_630w7d["family"].astype(str).to_numpy(),
    table_630w7d["analysis_family"].astype(str).to_numpy(),
):
    raise RuntimeError(
        "Manifest family and analysis_family are no longer identical."
    )


# =============================================================================
# 3) RECREATE ONLY THE PREVIOUSLY DEFINED GAIN-SIGN CLASSIFICATION
# =============================================================================

gain_630w7d = table_630w7d[
    "selective_mae_gain"
].to_numpy(
    dtype=np.float64
)

positive_630w7d = (
    gain_630w7d
    >
    1e-12
)

zero_630w7d = np.isclose(
    gain_630w7d,
    0.0,
    rtol=0.0,
    atol=1e-12,
)

negative_630w7d = (
    gain_630w7d
    <
    -1e-12
)

if not np.all(
    positive_630w7d
    |
    zero_630w7d
    |
    negative_630w7d
):
    raise RuntimeError(
        "Gain-sign partition failed."
    )

table_630w7d["_positive_gain"] = (
    positive_630w7d.astype(np.int64)
)

table_630w7d["_zero_gain"] = (
    zero_630w7d.astype(np.int64)
)

table_630w7d["_negative_gain"] = (
    negative_630w7d.astype(np.int64)
)


expected_positive_630w7d = int(
    FD004_CORRUPTION_OUTCOME_REPORT_630W7B[
        "overall_summary"
    ][
        "positive_gain_count"
    ]
)

expected_zero_630w7d = int(
    FD004_CORRUPTION_OUTCOME_REPORT_630W7B[
        "overall_summary"
    ][
        "zero_gain_count"
    ]
)

expected_negative_630w7d = int(
    FD004_CORRUPTION_OUTCOME_REPORT_630W7B[
        "overall_summary"
    ][
        "negative_gain_count"
    ]
)

if (
    int(positive_630w7d.sum())
    !=
    expected_positive_630w7d
):
    raise RuntimeError(
        "Positive-gain count no longer matches W-7B."
    )

if (
    int(zero_630w7d.sum())
    !=
    expected_zero_630w7d
):
    raise RuntimeError(
        "Zero-gain count no longer matches W-7B."
    )

if (
    int(negative_630w7d.sum())
    !=
    expected_negative_630w7d
):
    raise RuntimeError(
        "Negative-gain count no longer matches W-7B."
    )


# =============================================================================
# 4) COMMON GROUP-SUMMARY BUILDER
# =============================================================================

def _group_summary_630w7d(
    frame,
    grouping_columns,
):
    result = (
        frame
        .groupby(
            list(grouping_columns),
            sort=False,
            dropna=False,
        )
        .agg(
            scenario_count=(
                "scenario_id",
                "size",
            ),

            coverage_median=(
                "coverage",
                "median",
            ),

            coverage_mean=(
                "coverage",
                "mean",
            ),

            full_mae_median=(
                "full_mae",
                "median",
            ),

            full_mae_mean=(
                "full_mae",
                "mean",
            ),

            accepted_mae_median=(
                "accepted_mae",
                "median",
            ),

            accepted_mae_mean=(
                "accepted_mae",
                "mean",
            ),

            abstained_mae_mean=(
                "abstained_mae",
                "mean",
            ),

            selective_gain_min=(
                "selective_mae_gain",
                "min",
            ),

            selective_gain_median=(
                "selective_mae_gain",
                "median",
            ),

            selective_gain_mean=(
                "selective_mae_gain",
                "mean",
            ),

            selective_gain_max=(
                "selective_mae_gain",
                "max",
            ),

            relative_reduction_percent_mean=(
                "relative_mae_reduction_percent",
                "mean",
            ),

            positive_gain_count=(
                "_positive_gain",
                "sum",
            ),

            zero_gain_count=(
                "_zero_gain",
                "sum",
            ),

            negative_gain_count=(
                "_negative_gain",
                "sum",
            ),
        )
        .reset_index()
    )

    for count_col in (
        "scenario_count",
        "positive_gain_count",
        "zero_gain_count",
        "negative_gain_count",
    ):
        result[count_col] = (
            result[count_col]
            .astype(np.int64)
        )

    result[
        "positive_gain_rate"
    ] = (
        result[
            "positive_gain_count"
        ].astype(np.float64)
        /
        result[
            "scenario_count"
        ].astype(np.float64)
    )

    return result


# =============================================================================
# 5) FAMILY × SEVERITY SUMMARY
# =============================================================================

family_severity_630w7d = _group_summary_630w7d(
    table_630w7d,
    (
        "family",
        "severity",
    ),
)

family_order_630w7d = {
    "bias": 0,
    "noise": 1,
    "drift": 2,
    "dropout": 3,
    "stuck": 4,
}

severity_order_630w7d = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

family_severity_630w7d[
    "_family_order"
] = (
    family_severity_630w7d[
        "family"
    ]
    .map(
        family_order_630w7d
    )
)

family_severity_630w7d[
    "_severity_order"
] = (
    family_severity_630w7d[
        "severity"
    ]
    .map(
        severity_order_630w7d
    )
)

if family_severity_630w7d[
    "_family_order"
].isna().any():
    raise RuntimeError(
        "Unexpected family label in family×severity summary."
    )

if family_severity_630w7d[
    "_severity_order"
].isna().any():
    raise RuntimeError(
        "Unexpected severity label in family×severity summary."
    )

family_severity_630w7d = (
    family_severity_630w7d
    .sort_values(
        [
            "_family_order",
            "_severity_order",
        ],
        kind="mergesort",
    )
    .drop(
        columns=[
            "_family_order",
            "_severity_order",
        ]
    )
    .reset_index(drop=True)
)

if len(
    family_severity_630w7d
) != 15:
    raise RuntimeError(
        "Expected exactly 15 family×severity groups."
    )


# =============================================================================
# 6) FAMILY × SENSOR SUMMARY
# =============================================================================

family_sensor_630w7d = _group_summary_630w7d(
    table_630w7d,
    (
        "family",
        "sensor",
    ),
)


def _sensor_number_630w7d(
    value
):
    text = str(value)

    if not text.startswith(
        "sensor_"
    ):
        raise RuntimeError(
            "Unexpected sensor label: "
            + text
        )

    try:
        return int(
            text.split(
                "_",
                1,
            )[1]
        )
    except Exception as exc:
        raise RuntimeError(
            "Could not parse sensor number from "
            + text
        ) from exc


family_sensor_630w7d[
    "_family_order"
] = (
    family_sensor_630w7d[
        "family"
    ].map(
        family_order_630w7d
    )
)

family_sensor_630w7d[
    "_sensor_number"
] = (
    family_sensor_630w7d[
        "sensor"
    ].map(
        _sensor_number_630w7d
    )
)

if family_sensor_630w7d[
    "_family_order"
].isna().any():
    raise RuntimeError(
        "Unexpected family label in family×sensor summary."
    )

family_sensor_630w7d = (
    family_sensor_630w7d
    .sort_values(
        [
            "_family_order",
            "_sensor_number",
        ],
        kind="mergesort",
    )
    .drop(
        columns=[
            "_family_order",
            "_sensor_number",
        ]
    )
    .reset_index(drop=True)
)


expected_family_sensor_groups_630w7d = (
    17
    +
    17
    +
    17
    +
    21
    +
    21
)

if len(
    family_sensor_630w7d
) != expected_family_sensor_groups_630w7d:
    raise RuntimeError(
        "Unexpected family×sensor group count."
    )


# =============================================================================
# 7) EXACT NEGATIVE-GAIN STUCK SCENARIOS
# =============================================================================

negative_rows_630w7d = (
    table_630w7d.loc[
        table_630w7d[
            "_negative_gain"
        ]
        ==
        1
    ]
    .copy()
)

if len(
    negative_rows_630w7d
) != 16:
    raise RuntimeError(
        "Expected exactly 16 negative-gain scenarios."
    )

if set(
    negative_rows_630w7d[
        "family"
    ].astype(str)
) != {
    "stuck"
}:
    raise RuntimeError(
        "A negative-gain scenario exists outside the stuck family."
    )


negative_stuck_630w7d = (
    negative_rows_630w7d[
        [
            "scenario_id",
            "scenario_number",
            "sensor",
            "severity",
            "alpha",
            "requested_fraction",
            "coverage",
            "full_mae",
            "accepted_mae",
            "abstained_mae",
            "selective_mae_gain",
            "relative_mae_reduction_percent",
            "abstained_minus_accepted_mae",
        ]
    ]
    .sort_values(
        [
            "selective_mae_gain",
            "scenario_number",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


negative_stuck_by_severity_630w7d = (
    negative_stuck_630w7d
    .groupby(
        "severity",
        sort=False,
        dropna=False,
    )
    .agg(
        negative_scenario_count=(
            "scenario_id",
            "size",
        ),

        gain_min=(
            "selective_mae_gain",
            "min",
        ),

        gain_median=(
            "selective_mae_gain",
            "median",
        ),

        gain_mean=(
            "selective_mae_gain",
            "mean",
        ),

        relative_reduction_percent_mean=(
            "relative_mae_reduction_percent",
            "mean",
        ),
    )
    .reset_index()
)

negative_stuck_by_severity_630w7d[
    "negative_scenario_count"
] = (
    negative_stuck_by_severity_630w7d[
        "negative_scenario_count"
    ].astype(np.int64)
)

negative_stuck_by_severity_630w7d[
    "_severity_order"
] = (
    negative_stuck_by_severity_630w7d[
        "severity"
    ].map(
        severity_order_630w7d
    )
)

negative_stuck_by_severity_630w7d = (
    negative_stuck_by_severity_630w7d
    .sort_values(
        "_severity_order",
        kind="mergesort",
    )
    .drop(
        columns=[
            "_severity_order"
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# 8) FREEZE FULL REPORTING TABLES
# =============================================================================

FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_630W7D = (
    family_severity_630w7d.copy()
)

FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_630W7D = (
    family_sensor_630w7d.copy()
)

FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D = (
    negative_stuck_630w7d.copy()
)

FD004_NEGATIVE_GAIN_STUCK_BY_SEVERITY_630W7D = (
    negative_stuck_by_severity_630w7d.copy()
)


def _dataframe_records_sha256_630w7d(
    frame
):
    records = frame.to_dict(
        orient="records"
    )

    return hashlib.sha256(
        json.dumps(
            records,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_SHA256_630W7D = (
    _dataframe_records_sha256_630w7d(
        FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_630W7D
    )
)

FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_SHA256_630W7D = (
    _dataframe_records_sha256_630w7d(
        FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_630W7D
    )
)

FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_SHA256_630W7D = (
    _dataframe_records_sha256_630w7d(
        FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D
    )
)


# =============================================================================
# 9) COMPACT FAMILY×SENSOR DIAGNOSTIC VIEW
#
# Full family×sensor table remains frozen resident.
# Display only the lowest 5 and highest 3 sensor-level mean gains per family.
# =============================================================================

sensor_diagnostic_parts_630w7d = []

for family_630w7d in (
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
):
    subset = (
        family_sensor_630w7d.loc[
            family_sensor_630w7d[
                "family"
            ]
            ==
            family_630w7d
        ]
        .copy()
    )

    low = (
        subset
        .sort_values(
            [
                "selective_gain_mean",
                "sensor",
            ],
            ascending=[
                True,
                True,
            ],
            kind="mergesort",
        )
        .head(5)
        .copy()
    )

    low[
        "diagnostic_rank"
    ] = "LOWEST_GAIN"

    high = (
        subset
        .sort_values(
            [
                "selective_gain_mean",
                "sensor",
            ],
            ascending=[
                False,
                True,
            ],
            kind="mergesort",
        )
        .head(3)
        .copy()
    )

    high[
        "diagnostic_rank"
    ] = "HIGHEST_GAIN"

    sensor_diagnostic_parts_630w7d.extend(
        [
            low,
            high,
        ]
    )


FD004_CORRUPTION_SENSOR_DIAGNOSTIC_630W7D = (
    pd.concat(
        sensor_diagnostic_parts_630w7d,
        axis=0,
        ignore_index=True,
        sort=False,
    )
)


# =============================================================================
# 10) FREEZE SUMMARY CONTRACT
# =============================================================================

summary_contract_630w7d = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-7D",

    "status":
        "381_CORRUPTION_GROUP_SUMMARIES_FROZEN",

    "scenario_count":
        381,

    "family_severity_group_count":
        int(
            len(
                FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_630W7D
            )
        ),

    "family_sensor_group_count":
        int(
            len(
                FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_630W7D
            )
        ),

    "negative_gain_scenario_count":
        int(
            len(
                FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D
            )
        ),

    "negative_gain_families":
        tuple(
            sorted(
                set(
                    negative_rows_630w7d[
                        "family"
                    ].astype(str)
                )
            )
        ),

    "family_severity_summary_sha256":
        FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_SHA256_630W7D,

    "family_sensor_summary_sha256":
        FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_SHA256_630W7D,

    "negative_stuck_scenarios_sha256":
        FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_SHA256_630W7D,

    "threshold_modified":
        False,

    "policy_modified":
        False,

    "new_corruption_executed":
        False,

    "new_model_inference_performed":
        False,

    "new_mc_execution_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,
}


FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_630W7D = (
    summary_contract_630w7d
)

FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_SHA256_630W7D = (
    hashlib.sha256(
        json.dumps(
            summary_contract_630w7d,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w7d = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-7D — FD004 CORRUPTION FAMILY × "
    "SEVERITY / SENSOR SUMMARY"
)

print("=" * 176)

print("New predictions                    :", False)
print("New corruption execution           :", False)
print("Threshold modified                 :", False)
print("Policy modified                    :", False)
print("TEST contents read                 :", False)
print("Official RUL contents read         :", False)


print("\n--- FAMILY × SEVERITY SUMMARY ---")

display_columns_severity_630w7d = [
    "family",
    "severity",
    "scenario_count",
    "coverage_mean",
    "full_mae_mean",
    "accepted_mae_mean",
    "selective_gain_min",
    "selective_gain_median",
    "selective_gain_mean",
    "selective_gain_max",
    "relative_reduction_percent_mean",
    "positive_gain_count",
    "negative_gain_count",
    "positive_gain_rate",
]

print(
    FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_630W7D[
        display_columns_severity_630w7d
    ].to_string(
        index=False
    )
)


print("\n--- FAMILY × SENSOR DIAGNOSTIC ---")

print(
    "Full resident family×sensor groups:",
    len(
        FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_630W7D
    ),
)

print(
    "Displaying each family's 5 lowest and 3 highest "
    "sensor-level mean selective gains."
)

display_columns_sensor_630w7d = [
    "family",
    "sensor",
    "diagnostic_rank",
    "scenario_count",
    "coverage_mean",
    "full_mae_mean",
    "accepted_mae_mean",
    "selective_gain_min",
    "selective_gain_mean",
    "selective_gain_max",
    "positive_gain_count",
    "negative_gain_count",
]

print(
    FD004_CORRUPTION_SENSOR_DIAGNOSTIC_630W7D[
        display_columns_sensor_630w7d
    ].to_string(
        index=False
    )
)


print("\n--- NEGATIVE-GAIN STUCK SCENARIOS ---")

print(
    "Count                             :",
    len(
        FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D
    ),
)

print(
    FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D.to_string(
        index=False
    )
)


print("\n--- NEGATIVE STUCK BY SEVERITY ---")

print(
    FD004_NEGATIVE_GAIN_STUCK_BY_SEVERITY_630W7D.to_string(
        index=False
    )
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Family×severity groups            :",
    len(
        FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_630W7D
    ),
)

print(
    "Family×sensor groups              :",
    len(
        FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_630W7D
    ),
)

print(
    "Family×severity SHA-256           :",
    FD004_CORRUPTION_FAMILY_SEVERITY_SUMMARY_SHA256_630W7D,
)

print(
    "Family×sensor SHA-256             :",
    FD004_CORRUPTION_FAMILY_SENSOR_SUMMARY_SHA256_630W7D,
)

print(
    "Negative-stuck SHA-256            :",
    FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_SHA256_630W7D,
)

print(
    "Group-summary contract SHA-256    :",
    FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_SHA256_630W7D,
)

print(
    "\nPASS — frozen 381-scenario outcomes were summarized by "
    "family×severity and family×sensor using resident results only."
)

print(
    "All negative-gain scenarios were preserved explicitly; "
    "no threshold/policy retuning occurred and TEST/RUL remain sealed."
)

In [ ]:
# STEP 6.30W-8 — FD004 PRE-TEST EVIDENCE CLOSURE / RESTART CHECKPOINT
#
# Purpose:
# Freeze ALL pre-TEST scientific evidence before any held-out TEST/RUL access.
#
# Locks:
# - transferable methodology
# - frozen FD004 operating policy / threshold
# - clean DEV selective outcomes
# - all 381 corruption outcomes
# - family × severity / family × sensor summaries
# - explicit stuck-family limitation
#
# NO new model inference.
# NO new MC execution.
# NO corruption execution.
# NO threshold/policy modification.
# NO TEST access.
# NO official RUL access.

import os
import json
import hashlib
import numpy as np


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w8 = (
    "all_checks_passed_630w7d",

    "FD004_OPERATING_POLICY_METHODOLOGY_630W4",
    "FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4",

    "FD004_FROZEN_OPERATING_POLICY_630W5",
    "FD004_FROZEN_OPERATING_POLICY_SHA256_630W5",

    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B",
    "FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B",

    "FD004_CORRUPTION_OUTCOME_REPORT_630W7B",
    "FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B",

    "FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_630W7D",
    "FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_SHA256_630W7D",

    "FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D",
    "FD004_NEGATIVE_GAIN_STUCK_BY_SEVERITY_630W7D",

    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P",
    "FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P",
)

missing_630w8 = [
    name
    for name in required_630w8
    if name not in globals()
]

if missing_630w8:
    raise RuntimeError(
        "Missing STEP 6.30W-8 authority object(s): "
        + ", ".join(missing_630w8)
    )

if not bool(all_checks_passed_630w7d):
    raise RuntimeError(
        "STEP 6.30W-7D must remain passed."
    )


# =============================================================================
# 1) GENERIC CONTRACT-HASH VERIFIER
# =============================================================================

def _json_sha256_630w8(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


contracts_to_verify_630w8 = (
    (
        "methodology",
        FD004_OPERATING_POLICY_METHODOLOGY_630W4,
        FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4,
    ),
    (
        "frozen operating policy",
        FD004_FROZEN_OPERATING_POLICY_630W5,
        FD004_FROZEN_OPERATING_POLICY_SHA256_630W5,
    ),
    (
        "clean DEV outcome report",
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B,
        FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B,
    ),
    (
        "381 corruption outcome report",
        FD004_CORRUPTION_OUTCOME_REPORT_630W7B,
        FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B,
    ),
    (
        "corruption group-summary contract",
        FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_630W7D,
        FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_SHA256_630W7D,
    ),
    (
        "label-free 381 corruption closure",
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_630V18P,
        FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P,
    ),
)


contract_hash_checks_630w8 = {}

for (
    label_630w8,
    contract_630w8,
    expected_sha_630w8,
) in contracts_to_verify_630w8:

    actual_sha_630w8 = _json_sha256_630w8(
        contract_630w8
    )

    passed_630w8 = bool(
        actual_sha_630w8
        ==
        str(expected_sha_630w8)
    )

    contract_hash_checks_630w8[
        label_630w8
    ] = passed_630w8

    if not passed_630w8:
        raise RuntimeError(
            f"Frozen {label_630w8} hash verification failed."
        )


# =============================================================================
# 2) POLICY / LEAKAGE GUARDS
# =============================================================================

methodology_630w8 = (
    FD004_OPERATING_POLICY_METHODOLOGY_630W4
)

policy_630w8 = (
    FD004_FROZEN_OPERATING_POLICY_630W5
)

clean_report_630w8 = (
    FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_630W6B
)

corruption_report_630w8 = (
    FD004_CORRUPTION_OUTCOME_REPORT_630W7B
)

group_contract_630w8 = (
    FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_630W7D
)


if float(
    methodology_630w8[
        "nominal_coverage"
    ]
) != float(
    policy_630w8[
        "nominal_coverage"
    ]
):
    raise RuntimeError(
        "Methodology/policy nominal coverage mismatch."
    )

if float(
    policy_630w8[
        "frozen_risk_threshold"
    ]
) != float(
    clean_report_630w8[
        "frozen_risk_threshold"
    ]
):
    raise RuntimeError(
        "Frozen threshold differs between policy and CLEAN DEV report."
    )

if float(
    policy_630w8[
        "frozen_risk_threshold"
    ]
) != float(
    corruption_report_630w8[
        "frozen_risk_threshold"
    ]
):
    raise RuntimeError(
        "Frozen threshold differs between policy and corruption report."
    )


for forbidden_key_630w8 in (
    "dev_labels_used_for_threshold",
    "dev_errors_used_for_threshold",
    "corruption_predictions_used_for_threshold",
    "corruption_risks_used_for_threshold",
    "corruption_errors_used_for_threshold",
):
    if bool(
        policy_630w8.get(
            forbidden_key_630w8,
            False,
        )
    ):
        raise RuntimeError(
            "Policy leakage guard failed: "
            + forbidden_key_630w8
        )


if bool(
    clean_report_630w8[
        "threshold_modified"
    ]
):
    raise RuntimeError(
        "Threshold changed during CLEAN DEV reporting."
    )

if bool(
    clean_report_630w8[
        "policy_modified"
    ]
):
    raise RuntimeError(
        "Policy changed during CLEAN DEV reporting."
    )

if bool(
    corruption_report_630w8[
        "threshold_modified"
    ]
):
    raise RuntimeError(
        "Threshold changed during corruption reporting."
    )

if bool(
    corruption_report_630w8[
        "threshold_recomputed"
    ]
):
    raise RuntimeError(
        "Threshold was recomputed during corruption reporting."
    )

if bool(
    corruption_report_630w8[
        "policy_modified"
    ]
):
    raise RuntimeError(
        "Policy changed during corruption reporting."
    )


# =============================================================================
# 3) VERIFY MASTER COUNTS / NEGATIVE-GAIN EXCEPTION
# =============================================================================

overall_630w8 = (
    corruption_report_630w8[
        "overall_summary"
    ]
)

if int(
    overall_630w8[
        "scenario_count"
    ]
) != 381:
    raise RuntimeError(
        "Corruption scenario count is no longer 381."
    )

if (
    int(
        overall_630w8[
            "positive_gain_count"
        ]
    )
    +
    int(
        overall_630w8[
            "zero_gain_count"
        ]
    )
    +
    int(
        overall_630w8[
            "negative_gain_count"
        ]
    )
    !=
    381
):
    raise RuntimeError(
        "Gain-sign counts do not sum to 381."
    )


negative_stuck_630w8 = (
    FD004_NEGATIVE_GAIN_STUCK_SCENARIOS_630W7D
    .copy()
    .reset_index(drop=True)
)

if len(
    negative_stuck_630w8
) != int(
    overall_630w8[
        "negative_gain_count"
    ]
):
    raise RuntimeError(
        "Negative-scenario table count no longer matches W-7B."
    )

if len(
    negative_stuck_630w8
) != 16:
    raise RuntimeError(
        "Expected exactly 16 negative-gain scenarios."
    )


negative_by_severity_frame_630w8 = (
    FD004_NEGATIVE_GAIN_STUCK_BY_SEVERITY_630W7D
    .copy()
    .reset_index(drop=True)
)

negative_by_severity_630w8 = {
    str(row["severity"]):
        int(
            row[
                "negative_scenario_count"
            ]
        )
    for _, row
    in negative_by_severity_frame_630w8.iterrows()
}

if sum(
    negative_by_severity_630w8.values()
) != 16:
    raise RuntimeError(
        "Negative stuck severity counts do not sum to 16."
    )


# =============================================================================
# 4) SCIENTIFIC CONCLUSION — DERIVED ONLY FROM FROZEN RESULTS
# =============================================================================

clean_gain_positive_630w8 = bool(
    float(
        clean_report_630w8[
            "selective_mae_gain"
        ]
    )
    >
    0.0
)

corruption_majority_positive_630w8 = bool(
    int(
        overall_630w8[
            "positive_gain_count"
        ]
    )
    >
    int(
        overall_630w8[
            "negative_gain_count"
        ]
    )
)

all_negative_are_stuck_630w8 = bool(
    tuple(
        group_contract_630w8[
            "negative_gain_families"
        ]
    )
    ==
    (
        "stuck",
    )
)

if not all_negative_are_stuck_630w8:
    raise RuntimeError(
        "Negative-gain family evidence changed."
    )


scientific_conclusion_630w8 = (
    "The frozen FD004 reliability-aware selective policy improves CLEAN DEV "
    "MAE at approximately 80% realized coverage and yields positive selective "
    "MAE gain in the large majority of controlled corruption scenarios. "
    "Performance is not uniformly robust: all observed negative-gain "
    "corruption cases occur in the stuck-sensor family, with the limitation "
    "concentrated most strongly at high severity. No threshold or policy "
    "retuning is permitted after this pre-TEST closure."
)


# =============================================================================
# 5) PRE-TEST EVIDENCE CLOSURE
# =============================================================================

closure_payload_raw_630w8 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-8",

    "status":
        "PRETEST_EVIDENCE_LOCKED_READY_FOR_ONE_WAY_TEST_ADMISSION",

    "methodology_sha256":
        str(
            FD004_OPERATING_POLICY_METHODOLOGY_SHA256_630W4
        ),

    "frozen_policy_sha256":
        str(
            FD004_FROZEN_OPERATING_POLICY_SHA256_630W5
        ),

    "clean_dev_outcome_report_sha256":
        str(
            FD004_CLEAN_DEV_SELECTIVE_OUTCOME_REPORT_SHA256_630W6B
        ),

    "corruption_outcome_report_sha256":
        str(
            FD004_CORRUPTION_OUTCOME_REPORT_SHA256_630W7B
        ),

    "corruption_group_summary_sha256":
        str(
            FD004_CORRUPTION_GROUP_SUMMARY_CONTRACT_SHA256_630W7D
        ),

    "label_free_corruption_closure_sha256":
        str(
            FD004_CONTROLLED_CORRUPTION_381_CLOSURE_SHA256_630V18P
        ),

    "policy": {
        "policy_form":
            str(
                policy_630w8[
                    "policy_form"
                ]
            ),

        "nominal_coverage":
            float(
                policy_630w8[
                    "nominal_coverage"
                ]
            ),

        "frozen_risk_threshold":
            float(
                policy_630w8[
                    "frozen_risk_threshold"
                ]
            ),

        "acceptance_comparator":
            str(
                policy_630w8[
                    "acceptance_comparator"
                ]
            ),

        "acceptance_rule":
            str(
                policy_630w8[
                    "acceptance_rule"
                ]
            ),

        "tie_handling":
            str(
                policy_630w8[
                    "tie_handling"
                ]
            ),
    },

    "clean_dev": {
        "population":
            int(
                clean_report_630w8[
                    "population"
                ]
            ),

        "realized_coverage":
            float(
                clean_report_630w8[
                    "realized_coverage"
                ]
            ),

        "accepted_count":
            int(
                clean_report_630w8[
                    "accepted_count"
                ]
            ),

        "abstained_count":
            int(
                clean_report_630w8[
                    "abstained_count"
                ]
            ),

        "full_mae":
            float(
                clean_report_630w8[
                    "full_mae"
                ]
            ),

        "accepted_mae":
            float(
                clean_report_630w8[
                    "accepted_mae"
                ]
            ),

        "abstained_mae":
            float(
                clean_report_630w8[
                    "abstained_mae"
                ]
            ),

        "selective_mae_gain":
            float(
                clean_report_630w8[
                    "selective_mae_gain"
                ]
            ),

        "relative_mae_reduction_percent":
            float(
                clean_report_630w8[
                    "relative_mae_reduction_percent"
                ]
            ),
    },

    "controlled_corruption": {
        "scenario_count":
            381,

        "positive_gain_count":
            int(
                overall_630w8[
                    "positive_gain_count"
                ]
            ),

        "zero_gain_count":
            int(
                overall_630w8[
                    "zero_gain_count"
                ]
            ),

        "negative_gain_count":
            int(
                overall_630w8[
                    "negative_gain_count"
                ]
            ),

        "positive_gain_rate":
            float(
                overall_630w8[
                    "positive_gain_rate"
                ]
            ),

        "selective_gain_median":
            float(
                overall_630w8[
                    "selective_gain_median"
                ]
            ),

        "selective_gain_mean":
            float(
                overall_630w8[
                    "selective_gain_mean"
                ]
            ),

        "relative_reduction_percent_mean":
            float(
                overall_630w8[
                    "relative_reduction_percent_mean"
                ]
            ),

        "negative_gain_family":
            "stuck",

        "negative_gain_by_severity":
            negative_by_severity_630w8,
    },

    "scientific_conclusion":
        scientific_conclusion_630w8,

    "clean_dev_gain_positive":
        clean_gain_positive_630w8,

    "corruption_majority_positive":
        corruption_majority_positive_630w8,

    "all_negative_gain_cases_are_stuck":
        all_negative_are_stuck_630w8,

    "model_retrained_after_policy_freeze":
        False,

    "risk_architecture_modified_after_policy_freeze":
        False,

    "threshold_modified_after_policy_freeze":
        False,

    "threshold_recomputed_after_policy_freeze":
        False,

    "policy_modified_after_policy_freeze":
        False,

    "test_admission_performed":
        False,

    "test_contents_read":
        False,

    "official_rul_contents_read":
        False,

    "future_test_rule":
        (
            "one-way held-out TEST evaluation only; "
            "apply frozen preprocessing/model/risk/policy exactly; "
            "no threshold or policy tuning after TEST access"
        ),
}


# Canonicalize to JSON-native types before hashing/storing.
FD004_PRETEST_EVIDENCE_CLOSURE_630W8 = json.loads(
    json.dumps(
        closure_payload_raw_630w8,
        sort_keys=True,
        allow_nan=False,
    )
)


FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8 = (
    _json_sha256_630w8(
        FD004_PRETEST_EVIDENCE_CLOSURE_630W8
    )
)


# =============================================================================
# 6) RESTART-SAFE CHECKPOINT
# =============================================================================

checkpoint_path_630w8 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_pretest_evidence_closure_630w8_v1.json"
)

checkpoint_dir_630w8 = os.path.dirname(
    checkpoint_path_630w8
)

if checkpoint_dir_630w8:
    os.makedirs(
        checkpoint_dir_630w8,
        exist_ok=True,
    )


checkpoint_payload_630w8 = {
    "FD004_PRETEST_EVIDENCE_CLOSURE_630W8":
        FD004_PRETEST_EVIDENCE_CLOSURE_630W8,

    "FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8":
        FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8,
}


checkpoint_tmp_path_630w8 = (
    checkpoint_path_630w8
    +
    ".tmp"
)

with open(
    checkpoint_tmp_path_630w8,
    "w",
    encoding="utf-8",
) as handle_630w8:
    json.dump(
        checkpoint_payload_630w8,
        handle_630w8,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )

os.replace(
    checkpoint_tmp_path_630w8,
    checkpoint_path_630w8,
)


with open(
    checkpoint_path_630w8,
    "rb",
) as handle_630w8:
    checkpoint_bytes_630w8 = (
        handle_630w8.read()
    )

FD004_PRETEST_CHECKPOINT_FILE_SHA256_630W8 = (
    hashlib.sha256(
        checkpoint_bytes_630w8
    ).hexdigest()
)


with open(
    checkpoint_path_630w8,
    "r",
    encoding="utf-8",
) as handle_630w8:
    checkpoint_roundtrip_630w8 = (
        json.load(
            handle_630w8
        )
    )


checkpoint_roundtrip_exact_630w8 = bool(
    checkpoint_roundtrip_630w8
    ==
    checkpoint_payload_630w8
)

if not checkpoint_roundtrip_exact_630w8:
    raise RuntimeError(
        "Pre-TEST checkpoint JSON roundtrip failed."
    )


# =============================================================================
# 7) FINAL CLOSURE FLAGS
# =============================================================================

FD004_PRETEST_EVIDENCE_LOCKED_630W8 = True
FD004_READY_FOR_ONE_WAY_TEST_ADMISSION_630W8 = True

FD004_THRESHOLD_MODIFIED_AFTER_PRETEST_CLOSURE_630W8 = False
FD004_POLICY_MODIFIED_AFTER_PRETEST_CLOSURE_630W8 = False

FD004_TEST_ADMISSION_PERFORMED_630W8 = False
FD004_TEST_CONTENTS_READ_630W8 = False
FD004_RUL_CONTENTS_READ_630W8 = False

all_checks_passed_630w8 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-8 — FD004 PRE-TEST EVIDENCE "
    "CLOSURE / RESTART CHECKPOINT"
)

print("=" * 176)

print("New model inference               :", False)
print("New MC execution                  :", False)
print("New corruption execution          :", False)
print("Threshold modified                :", False)
print("Threshold recomputed              :", False)
print("Policy modified                   :", False)
print("TEST admission performed          :", False)
print("TEST contents read                :", False)
print("Official RUL contents read        :", False)


print("\n--- FROZEN OPERATING POLICY ---")

print(
    "Nominal coverage                  :",
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "policy"
    ][
        "nominal_coverage"
    ],
)

print(
    "Frozen risk threshold             :",
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "policy"
    ][
        "frozen_risk_threshold"
    ],
)

print(
    "Acceptance rule                   :",
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "policy"
    ][
        "acceptance_rule"
    ],
)


print("\n--- CLEAN DEV FROZEN RESULT ---")

clean_630w8 = (
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "clean_dev"
    ]
)

print(
    "Coverage                          :",
    clean_630w8[
        "realized_coverage"
    ],
)

print(
    "Full / accepted / abstained MAE   :",
    (
        clean_630w8[
            "full_mae"
        ],
        clean_630w8[
            "accepted_mae"
        ],
        clean_630w8[
            "abstained_mae"
        ],
    ),
)

print(
    "Selective MAE gain                :",
    clean_630w8[
        "selective_mae_gain"
    ],
)

print(
    "Relative MAE reduction (%)        :",
    clean_630w8[
        "relative_mae_reduction_percent"
    ],
)


print("\n--- 381-SCENARIO FROZEN RESULT ---")

corr_630w8 = (
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "controlled_corruption"
    ]
)

print(
    "Positive / zero / negative gain   :",
    (
        corr_630w8[
            "positive_gain_count"
        ],
        corr_630w8[
            "zero_gain_count"
        ],
        corr_630w8[
            "negative_gain_count"
        ],
    ),
)

print(
    "Positive-gain rate                :",
    corr_630w8[
        "positive_gain_rate"
    ],
)

print(
    "Negative-gain family              :",
    corr_630w8[
        "negative_gain_family"
    ],
)

print(
    "Negative stuck by severity        :",
    corr_630w8[
        "negative_gain_by_severity"
    ],
)


print("\n--- SCIENTIFIC CONCLUSION ---")

print(
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8[
        "scientific_conclusion"
    ]
)


print("\n--- EVIDENCE LOCK ---")

print(
    "All source contract hashes exact  :",
    all(
        contract_hash_checks_630w8.values()
    ),
)

print(
    "Pre-TEST closure SHA-256          :",
    FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8,
)

print(
    "Checkpoint path                   :",
    checkpoint_path_630w8,
)

print(
    "Checkpoint file SHA-256           :",
    FD004_PRETEST_CHECKPOINT_FILE_SHA256_630W8,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_630w8,
)

print(
    "Ready for one-way TEST admission  :",
    FD004_READY_FOR_ONE_WAY_TEST_ADMISSION_630W8,
)


print(
    "\nPASS — FD004 pre-TEST evidence is now closed and restart-safe."
)

print(
    "From this point onward the model, preprocessing, risk construction, "
    "threshold, and acceptance policy are immutable."
)

print(
    "TEST and official RUL remain sealed. The next stage is a separate "
    "one-way held-out TEST admission gate."
)

In [ ]:
# STEP 6.30W-9 — FD004 ONE-WAY HELD-OUT TEST ADMISSION GATE
#
# This is an ADMISSION GATE ONLY.
#
# It DOES:
# - verify the W-8 pre-TEST closure
# - verify the restart checkpoint currently on disk
# - assert that preprocessing/model/risk/threshold/policy are immutable
# - authorize the NEXT stage to read held-out TEST + official RUL exactly
#   for one-way evaluation
# - create a restart-safe admission contract
#
# It DOES NOT:
# - open TEST
# - open official RUL
# - run inference
# - compute TEST predictions/risk/errors
# - modify threshold/policy/model/preprocessing
#
# TEST/RUL contents remain sealed after this cell.

import os
import json
import hashlib


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w9 = (
    "all_checks_passed_630w8",

    "FD004_PRETEST_EVIDENCE_CLOSURE_630W8",
    "FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8",

    "FD004_PRETEST_CHECKPOINT_FILE_SHA256_630W8",

    "FD004_PRETEST_EVIDENCE_LOCKED_630W8",
    "FD004_READY_FOR_ONE_WAY_TEST_ADMISSION_630W8",

    "FD004_THRESHOLD_MODIFIED_AFTER_PRETEST_CLOSURE_630W8",
    "FD004_POLICY_MODIFIED_AFTER_PRETEST_CLOSURE_630W8",

    "FD004_TEST_ADMISSION_PERFORMED_630W8",
    "FD004_TEST_CONTENTS_READ_630W8",
    "FD004_RUL_CONTENTS_READ_630W8",
)

missing_630w9 = [
    name
    for name in required_630w9
    if name not in globals()
]

if missing_630w9:
    raise RuntimeError(
        "Missing STEP 6.30W-9 authority object(s): "
        + ", ".join(missing_630w9)
    )

if not bool(all_checks_passed_630w8):
    raise RuntimeError(
        "STEP 6.30W-8 must remain passed."
    )

if not bool(
    FD004_PRETEST_EVIDENCE_LOCKED_630W8
):
    raise RuntimeError(
        "FD004 pre-TEST evidence is not marked locked."
    )

if not bool(
    FD004_READY_FOR_ONE_WAY_TEST_ADMISSION_630W8
):
    raise RuntimeError(
        "FD004 is not marked ready for one-way TEST admission."
    )


# =============================================================================
# 1) VERIFY PRE-TEST CLOSURE HASH
# =============================================================================

def _json_sha256_630w9(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


closure_sha_recomputed_630w9 = _json_sha256_630w9(
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8
)

closure_sha_exact_630w9 = bool(
    closure_sha_recomputed_630w9
    ==
    str(
        FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8
    )
)

if not closure_sha_exact_630w9:
    raise RuntimeError(
        "W-8 pre-TEST evidence closure changed."
    )


# =============================================================================
# 2) VERIFY W-8 RESTART CHECKPOINT ON DISK
# =============================================================================

pretest_checkpoint_path_630w9 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_pretest_evidence_closure_630w8_v1.json"
)

if not os.path.isfile(
    pretest_checkpoint_path_630w9
):
    raise RuntimeError(
        "W-8 pre-TEST restart checkpoint is missing from disk."
    )


with open(
    pretest_checkpoint_path_630w9,
    "rb",
) as handle_630w9:
    pretest_checkpoint_bytes_630w9 = (
        handle_630w9.read()
    )


pretest_checkpoint_file_sha_630w9 = hashlib.sha256(
    pretest_checkpoint_bytes_630w9
).hexdigest()

pretest_checkpoint_file_sha_exact_630w9 = bool(
    pretest_checkpoint_file_sha_630w9
    ==
    str(
        FD004_PRETEST_CHECKPOINT_FILE_SHA256_630W8
    )
)

if not pretest_checkpoint_file_sha_exact_630w9:
    raise RuntimeError(
        "W-8 pre-TEST checkpoint file SHA changed."
    )


with open(
    pretest_checkpoint_path_630w9,
    "r",
    encoding="utf-8",
) as handle_630w9:
    pretest_checkpoint_payload_630w9 = json.load(
        handle_630w9
    )


checkpoint_closure_630w9 = (
    pretest_checkpoint_payload_630w9.get(
        "FD004_PRETEST_EVIDENCE_CLOSURE_630W8"
    )
)

checkpoint_closure_sha_630w9 = (
    pretest_checkpoint_payload_630w9.get(
        "FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8"
    )
)


checkpoint_payload_exact_630w9 = bool(
    checkpoint_closure_630w9
    ==
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8
    and
    str(
        checkpoint_closure_sha_630w9
    )
    ==
    str(
        FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8
    )
)

if not checkpoint_payload_exact_630w9:
    raise RuntimeError(
        "W-8 checkpoint payload no longer matches resident closure."
    )


# =============================================================================
# 3) VERIFY THE GATE IS ACTUALLY PRE-TEST
# =============================================================================

if bool(
    FD004_TEST_ADMISSION_PERFORMED_630W8
):
    raise RuntimeError(
        "W-8 already indicates TEST admission was performed."
    )

if bool(
    FD004_TEST_CONTENTS_READ_630W8
):
    raise RuntimeError(
        "W-8 indicates TEST contents were already read."
    )

if bool(
    FD004_RUL_CONTENTS_READ_630W8
):
    raise RuntimeError(
        "W-8 indicates official RUL contents were already read."
    )

if bool(
    FD004_THRESHOLD_MODIFIED_AFTER_PRETEST_CLOSURE_630W8
):
    raise RuntimeError(
        "Threshold modification detected after W-8 closure."
    )

if bool(
    FD004_POLICY_MODIFIED_AFTER_PRETEST_CLOSURE_630W8
):
    raise RuntimeError(
        "Policy modification detected after W-8 closure."
    )


closure_630w9 = (
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8
)

for key_630w9 in (
    "model_retrained_after_policy_freeze",
    "risk_architecture_modified_after_policy_freeze",
    "threshold_modified_after_policy_freeze",
    "threshold_recomputed_after_policy_freeze",
    "policy_modified_after_policy_freeze",
):
    if bool(
        closure_630w9[
            key_630w9
        ]
    ):
        raise RuntimeError(
            "Pre-TEST immutability guard failed: "
            + key_630w9
        )

if bool(
    closure_630w9[
        "test_contents_read"
    ]
):
    raise RuntimeError(
        "Pre-TEST closure says TEST contents were read."
    )

if bool(
    closure_630w9[
        "official_rul_contents_read"
    ]
):
    raise RuntimeError(
        "Pre-TEST closure says official RUL contents were read."
    )


# =============================================================================
# 4) FREEZE ONE-WAY TEST ADMISSION RULE
# =============================================================================

FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-9",

    "status":
        "ONE_WAY_HELDOUT_TEST_ADMITTED_CONTENTS_STILL_UNREAD",

    "pretest_evidence_closure_sha256":
        str(
            FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8
        ),

    "pretest_checkpoint_file_sha256":
        str(
            FD004_PRETEST_CHECKPOINT_FILE_SHA256_630W8
        ),

    "pretest_checkpoint_payload_exact":
        True,

    "test_admission_performed":
        True,

    "test_contents_read_in_gate":
        False,

    "official_rul_contents_read_in_gate":
        False,

    "authorized_next_stage":
        "one_way_heldout_test_evaluation",

    "authorized_data_access":
        (
            "FD004 held-out TEST inputs and official RUL labels "
            "may be opened only for fixed final evaluation"
        ),

    "preprocessing_immutable":
        True,

    "feature_rule_immutable":
        True,

    "window_rule_immutable":
        True,

    "model_immutable":
        True,

    "mc_dropout_protocol_immutable":
        True,

    "risk_construction_immutable":
        True,

    "risk_calibration_immutable":
        True,

    "frozen_risk_threshold":
        float(
            closure_630w9[
                "policy"
            ][
                "frozen_risk_threshold"
            ]
        ),

    "threshold_immutable":
        True,

    "acceptance_rule":
        str(
            closure_630w9[
                "policy"
            ][
                "acceptance_rule"
            ]
        ),

    "acceptance_policy_immutable":
        True,

    "test_tuning_permitted":
        False,

    "threshold_reselection_after_test":
        False,

    "policy_reselection_after_test":
        False,

    "model_retraining_after_test":
        False,

    "preprocessing_refit_after_test":
        False,

    "risk_recalibration_after_test":
        False,

    "test_evaluation_rule":
        (
            "Open held-out FD004 TEST and official RUL once; "
            "apply frozen TRAIN/FIT-derived preprocessing, frozen model, "
            "frozen MC-dropout protocol, frozen risk construction/calibration, "
            "and frozen threshold/policy exactly; report full, accepted, "
            "and abstained endpoint MAE and coverage; do not tune anything "
            "using TEST outcomes."
        ),

    "test_result_interpretation_rule":
        (
            "Report the held-out result as observed, whether selective "
            "performance improves, is neutral, or worsens."
        ),
}


FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9 = (
    _json_sha256_630w9(
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9
    )
)


# =============================================================================
# 5) RESTART-SAFE TEST-ADMISSION CHECKPOINT
# =============================================================================

test_admission_checkpoint_path_630w9 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_one_way_test_admission_630w9_v1.json"
)

test_admission_checkpoint_dir_630w9 = os.path.dirname(
    test_admission_checkpoint_path_630w9
)

if test_admission_checkpoint_dir_630w9:
    os.makedirs(
        test_admission_checkpoint_dir_630w9,
        exist_ok=True,
    )


test_admission_checkpoint_payload_630w9 = {
    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9":
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9,

    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9":
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9,
}


test_admission_tmp_path_630w9 = (
    test_admission_checkpoint_path_630w9
    +
    ".tmp"
)

with open(
    test_admission_tmp_path_630w9,
    "w",
    encoding="utf-8",
) as handle_630w9:
    json.dump(
        test_admission_checkpoint_payload_630w9,
        handle_630w9,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )

os.replace(
    test_admission_tmp_path_630w9,
    test_admission_checkpoint_path_630w9,
)


with open(
    test_admission_checkpoint_path_630w9,
    "rb",
) as handle_630w9:
    admission_checkpoint_bytes_630w9 = (
        handle_630w9.read()
    )


FD004_TEST_ADMISSION_CHECKPOINT_FILE_SHA256_630W9 = (
    hashlib.sha256(
        admission_checkpoint_bytes_630w9
    ).hexdigest()
)


with open(
    test_admission_checkpoint_path_630w9,
    "r",
    encoding="utf-8",
) as handle_630w9:
    admission_roundtrip_630w9 = json.load(
        handle_630w9
    )


admission_roundtrip_exact_630w9 = bool(
    admission_roundtrip_630w9
    ==
    test_admission_checkpoint_payload_630w9
)

if not admission_roundtrip_exact_630w9:
    raise RuntimeError(
        "TEST-admission checkpoint JSON roundtrip failed."
    )


# =============================================================================
# 6) AUTHORITATIVE POST-GATE FLAGS
# =============================================================================

FD004_TEST_ADMISSION_PERFORMED_630W9 = True

FD004_TEST_EVALUATION_AUTHORIZED_630W9 = True

# IMPORTANT:
# Admission != data access.
FD004_TEST_CONTENTS_READ_630W9 = False
FD004_RUL_CONTENTS_READ_630W9 = False

FD004_THRESHOLD_MODIFIED_AFTER_TEST_ADMISSION_630W9 = False
FD004_THRESHOLD_RECOMPUTED_AFTER_TEST_ADMISSION_630W9 = False
FD004_POLICY_MODIFIED_AFTER_TEST_ADMISSION_630W9 = False

FD004_MODEL_RETRAINED_AFTER_TEST_ADMISSION_630W9 = False
FD004_PREPROCESSING_REFIT_AFTER_TEST_ADMISSION_630W9 = False
FD004_RISK_RECALIBRATED_AFTER_TEST_ADMISSION_630W9 = False

all_checks_passed_630w9 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-9 — FD004 ONE-WAY HELD-OUT TEST ADMISSION GATE"
)

print("=" * 176)

print(
    "Pre-TEST closure SHA exact        :",
    closure_sha_exact_630w9,
)

print(
    "Pre-TEST checkpoint SHA exact     :",
    pretest_checkpoint_file_sha_exact_630w9,
)

print(
    "Pre-TEST checkpoint payload exact :",
    checkpoint_payload_exact_630w9,
)


print("\n--- IMMUTABILITY GATE ---")

print("Preprocessing immutable           :", True)
print("Feature rule immutable            :", True)
print("Window rule immutable             :", True)
print("Model immutable                   :", True)
print("MC-dropout protocol immutable     :", True)
print("Risk construction immutable       :", True)
print("Risk calibration immutable        :", True)
print("Threshold immutable               :", True)
print("Acceptance policy immutable       :", True)

print(
    "Frozen risk threshold             :",
    FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9[
        "frozen_risk_threshold"
    ],
)

print(
    "Acceptance rule                   :",
    FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9[
        "acceptance_rule"
    ],
)


print("\n--- ONE-WAY TEST ADMISSION ---")

print(
    "TEST admission performed          :",
    FD004_TEST_ADMISSION_PERFORMED_630W9,
)

print(
    "TEST evaluation authorized        :",
    FD004_TEST_EVALUATION_AUTHORIZED_630W9,
)

print(
    "TEST contents read                :",
    FD004_TEST_CONTENTS_READ_630W9,
)

print(
    "Official RUL contents read        :",
    FD004_RUL_CONTENTS_READ_630W9,
)

print(
    "TEST tuning permitted             :",
    FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9[
        "test_tuning_permitted"
    ],
)


print("\n--- POST-ADMISSION GUARDS ---")

print(
    "Threshold modified                :",
    FD004_THRESHOLD_MODIFIED_AFTER_TEST_ADMISSION_630W9,
)

print(
    "Threshold recomputed              :",
    FD004_THRESHOLD_RECOMPUTED_AFTER_TEST_ADMISSION_630W9,
)

print(
    "Policy modified                   :",
    FD004_POLICY_MODIFIED_AFTER_TEST_ADMISSION_630W9,
)

print(
    "Model retrained                   :",
    FD004_MODEL_RETRAINED_AFTER_TEST_ADMISSION_630W9,
)

print(
    "Preprocessing refit               :",
    FD004_PREPROCESSING_REFIT_AFTER_TEST_ADMISSION_630W9,
)

print(
    "Risk recalibrated                 :",
    FD004_RISK_RECALIBRATED_AFTER_TEST_ADMISSION_630W9,
)


print("\n--- EVIDENCE LOCK ---")

print(
    "Admission contract SHA-256        :",
    FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9,
)

print(
    "Admission checkpoint path         :",
    test_admission_checkpoint_path_630w9,
)

print(
    "Admission checkpoint file SHA-256 :",
    FD004_TEST_ADMISSION_CHECKPOINT_FILE_SHA256_630W9,
)

print(
    "Admission checkpoint roundtrip    :",
    admission_roundtrip_exact_630w9,
)


print(
    "\nPASS — FD004 held-out TEST evaluation is now explicitly admitted."
)

print(
    "TEST and official RUL contents are STILL UNREAD in this gate."
)

print(
    "The next cell may perform the one-way held-out TEST evaluation "
    "under the frozen preprocessing/model/risk/threshold/policy only."
)

print(
    "After TEST access, no threshold, policy, model, preprocessing, "
    "or risk-calibration tuning is permitted regardless of the result."
)

In [ ]:
# STEP 6.30W-10A — FD004 FIRST HELD-OUT TEST/RUL ACCESS + INPUT BIND
#
# THIS IS THE FIRST ACTUAL ACCESS TO FD004 TEST + OFFICIAL RUL.
#
# Authorized by W-9 one-way admission.
#
# This cell DOES:
# - verify W-9 admission
# - open TEST_FD004.txt
# - open RUL_FD004.txt
# - hash both files
# - validate raw TEST structure/order
# - validate frozen regime-assignment rule against resident DEV authority
# - assign TEST regimes using the frozen canonical centers
# - apply frozen FD004 normalizer
# - build frozen 30-cycle TEST windows
# - identify exactly one final endpoint window per TEST engine
# - bind official RUL values to TEST engine order
# - save a restart-safe TEST-input binding
#
# This cell DOES NOT:
# - run deterministic model inference
# - run MC dropout
# - calculate selective risk
# - calculate TEST prediction errors
# - calculate TEST MAE
# - modify threshold/policy/preprocessing/model/calibration
#
# From this cell onward, TEST/RUL are considered OPENED.
# NO tuning is permitted regardless of any later result.

import os
import json
import hashlib
import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w10a = (
    "all_checks_passed_630w9",

    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9",
    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9",

    "FD004_TEST_ADMISSION_CHECKPOINT_FILE_SHA256_630W9",

    "FD004_TEST_ADMISSION_PERFORMED_630W9",
    "FD004_TEST_EVALUATION_AUTHORIZED_630W9",

    "FD004_TEST_CONTENTS_READ_630W9",
    "FD004_RUL_CONTENTS_READ_630W9",

    "FD004_THRESHOLD_MODIFIED_AFTER_TEST_ADMISSION_630W9",
    "FD004_THRESHOLD_RECOMPUTED_AFTER_TEST_ADMISSION_630W9",
    "FD004_POLICY_MODIFIED_AFTER_TEST_ADMISSION_630W9",
    "FD004_MODEL_RETRAINED_AFTER_TEST_ADMISSION_630W9",
    "FD004_PREPROCESSING_REFIT_AFTER_TEST_ADMISSION_630W9",
    "FD004_RISK_RECALIBRATED_AFTER_TEST_ADMISSION_630W9",

    "FD004_CANONICAL_REGIME_CENTERS_630F",
    "FD004_DEV_REGIME_IDS_630F",
    "FD004_DEV_CORRUPTION_BASE_630V17A",

    "normalize_fd004_raw_630v2",
    "build_windows_fd004_630v2",
    "sha256_array_630v2",
)

missing_630w10a = [
    name
    for name in required_630w10a
    if name not in globals()
]

if missing_630w10a:
    raise RuntimeError(
        "Missing STEP 6.30W-10A authority object(s): "
        + ", ".join(missing_630w10a)
    )

if not bool(all_checks_passed_630w9):
    raise RuntimeError(
        "STEP 6.30W-9 must remain passed."
    )

if not bool(
    FD004_TEST_ADMISSION_PERFORMED_630W9
):
    raise RuntimeError(
        "One-way TEST admission was not performed."
    )

if not bool(
    FD004_TEST_EVALUATION_AUTHORIZED_630W9
):
    raise RuntimeError(
        "Held-out TEST evaluation is not authorized."
    )

if bool(
    FD004_TEST_CONTENTS_READ_630W9
):
    raise RuntimeError(
        "W-9 unexpectedly says TEST contents were already read."
    )

if bool(
    FD004_RUL_CONTENTS_READ_630W9
):
    raise RuntimeError(
        "W-9 unexpectedly says official RUL contents were already read."
    )


# =============================================================================
# 1) VERIFY W-9 ADMISSION CONTRACT
# =============================================================================

def _json_sha256_630w10a(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


admission_sha_recomputed_630w10a = (
    _json_sha256_630w10a(
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9
    )
)

admission_sha_exact_630w10a = bool(
    admission_sha_recomputed_630w10a
    ==
    str(
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9
    )
)

if not admission_sha_exact_630w10a:
    raise RuntimeError(
        "W-9 TEST-admission contract changed."
    )


post_admission_guard_names_630w10a = (
    "FD004_THRESHOLD_MODIFIED_AFTER_TEST_ADMISSION_630W9",
    "FD004_THRESHOLD_RECOMPUTED_AFTER_TEST_ADMISSION_630W9",
    "FD004_POLICY_MODIFIED_AFTER_TEST_ADMISSION_630W9",
    "FD004_MODEL_RETRAINED_AFTER_TEST_ADMISSION_630W9",
    "FD004_PREPROCESSING_REFIT_AFTER_TEST_ADMISSION_630W9",
    "FD004_RISK_RECALIBRATED_AFTER_TEST_ADMISSION_630W9",
)

for guard_name_630w10a in post_admission_guard_names_630w10a:
    if bool(
        globals()[
            guard_name_630w10a
        ]
    ):
        raise RuntimeError(
            "Post-admission immutability guard failed: "
            + guard_name_630w10a
        )


# =============================================================================
# 2) RESOLVE EXACT FD004 TEST / RUL PATHS
# =============================================================================

expected_test_path_630w10a = (
    r"C:\RUL_Reliability_Research\02_dataset\raw"
    r"\CMAPSSData\test_FD004.txt"
)

expected_rul_path_630w10a = (
    r"C:\RUL_Reliability_Research\02_dataset\raw"
    r"\CMAPSSData\RUL_FD004.txt"
)


def _discover_matching_path_630w10a(
    basename,
    fallback_path,
):
    matches = []

    for name, value in list(
        globals().items()
    ):
        if not isinstance(
            value,
            str,
        ):
            continue

        normalized = value.replace(
            "/",
            "\\",
        )

        if normalized.lower().endswith(
            "\\" + basename.lower()
        ):
            matches.append(
                value
            )

    matches = tuple(
        dict.fromkeys(
            matches
        )
    )

    if matches:
        existing = tuple(
            path
            for path in matches
            if os.path.isfile(
                path
            )
        )

        if len(existing) == 1:
            return existing[0]

        if len(existing) > 1:
            normalized_existing = {
                os.path.normcase(
                    os.path.normpath(
                        path
                    )
                )
                for path in existing
            }

            if len(
                normalized_existing
            ) == 1:
                return existing[0]

            raise RuntimeError(
                f"Multiple distinct resident paths found for {basename}: "
                + repr(existing)
            )

    return fallback_path


test_path_630w10a = (
    _discover_matching_path_630w10a(
        "test_FD004.txt",
        expected_test_path_630w10a,
    )
)

rul_path_630w10a = (
    _discover_matching_path_630w10a(
        "RUL_FD004.txt",
        expected_rul_path_630w10a,
    )
)

if not os.path.isfile(
    test_path_630w10a
):
    raise RuntimeError(
        "FD004 TEST file not found at resolved path: "
        + test_path_630w10a
    )

if not os.path.isfile(
    rul_path_630w10a
):
    raise RuntimeError(
        "FD004 official RUL file not found at resolved path: "
        + rul_path_630w10a
    )


# =============================================================================
# 3) FIRST ACTUAL FILE ACCESS
#
# IMPORTANT:
# Set access flags immediately after bytes are read so that even if a later
# validation fails, the notebook truthfully records that TEST/RUL were opened.
# =============================================================================

FD004_TEST_CONTENTS_READ_630W10A = False
FD004_RUL_CONTENTS_READ_630W10A = False


with open(
    test_path_630w10a,
    "rb",
) as handle_630w10a:
    test_file_bytes_630w10a = (
        handle_630w10a.read()
    )

FD004_TEST_CONTENTS_READ_630W10A = True


with open(
    rul_path_630w10a,
    "rb",
) as handle_630w10a:
    rul_file_bytes_630w10a = (
        handle_630w10a.read()
    )

FD004_RUL_CONTENTS_READ_630W10A = True


FD004_TEST_FILE_SHA256_630W10A = (
    hashlib.sha256(
        test_file_bytes_630w10a
    ).hexdigest()
)

FD004_RUL_FILE_SHA256_630W10A = (
    hashlib.sha256(
        rul_file_bytes_630w10a
    ).hexdigest()
)


# =============================================================================
# 4) RECOVER AUTHORITATIVE RAW COLUMN SCHEMA FROM RESIDENT FD004 TRAIN
# =============================================================================

train_schema_candidates_630w10a = []

for name_630w10a, value_630w10a in list(
    globals().items()
):
    if not isinstance(
        value_630w10a,
        pd.DataFrame,
    ):
        continue

    if value_630w10a.shape == (
        61249,
        26,
    ):
        train_schema_candidates_630w10a.append(
            (
                name_630w10a,
                tuple(
                    value_630w10a.columns.tolist()
                ),
            )
        )


distinct_train_schemas_630w10a = tuple(
    dict.fromkeys(
        schema
        for _, schema
        in train_schema_candidates_630w10a
    )
)

if len(
    distinct_train_schemas_630w10a
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one resident FD004 raw 26-column "
        "TRAIN schema. Found distinct schema count: "
        + str(
            len(
                distinct_train_schemas_630w10a
            )
        )
    )

raw_columns_630w10a = (
    distinct_train_schemas_630w10a[
        0
    ]
)

if len(
    raw_columns_630w10a
) != 26:
    raise RuntimeError(
        "Recovered raw FD004 schema is not 26 columns."
    )


unit_col_630w10a = (
    raw_columns_630w10a[
        0
    ]
)

cycle_col_630w10a = (
    raw_columns_630w10a[
        1
    ]
)

op_cols_630w10a = (
    raw_columns_630w10a[
        2:5
    ]
)

sensor_cols_630w10a = (
    raw_columns_630w10a[
        5:26
    ]
)

if len(
    op_cols_630w10a
) != 3:
    raise RuntimeError(
        "Recovered operating-setting column count is not 3."
    )

if len(
    sensor_cols_630w10a
) != 21:
    raise RuntimeError(
        "Recovered sensor column count is not 21."
    )


# =============================================================================
# 5) PARSE TEST + OFFICIAL RUL
# =============================================================================

test_raw_630w10a = pd.read_csv(
    test_path_630w10a,
    sep=r"\s+",
    header=None,
    engine="python",
)

if test_raw_630w10a.shape[1] != 26:
    raise RuntimeError(
        "Parsed FD004 TEST does not contain exactly 26 columns. "
        f"Observed shape: {test_raw_630w10a.shape}"
    )

test_raw_630w10a.columns = list(
    raw_columns_630w10a
)


rul_raw_630w10a = pd.read_csv(
    rul_path_630w10a,
    sep=r"\s+",
    header=None,
    engine="python",
)

if rul_raw_630w10a.shape[1] != 1:
    raise RuntimeError(
        "Parsed FD004 official RUL file does not contain exactly one column. "
        f"Observed shape: {rul_raw_630w10a.shape}"
    )

official_rul_630w10a = (
    rul_raw_630w10a
    .iloc[
        :,
        0
    ]
    .to_numpy(
        dtype=np.float64
    )
    .reshape(-1)
)


# =============================================================================
# 6) RAW TEST INTEGRITY CHECKS
# =============================================================================

numeric_test_630w10a = (
    test_raw_630w10a
    .to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    numeric_test_630w10a
).all():
    raise RuntimeError(
        "FD004 TEST contains non-finite raw values."
    )

if not np.isfinite(
    official_rul_630w10a
).all():
    raise RuntimeError(
        "FD004 official RUL contains non-finite values."
    )

if np.any(
    official_rul_630w10a
    <
    0.0
):
    raise RuntimeError(
        "FD004 official RUL contains negative values."
    )


test_unit_ids_raw_630w10a = (
    test_raw_630w10a[
        unit_col_630w10a
    ]
    .to_numpy(
        dtype=np.int64
    )
)

test_cycles_raw_630w10a = (
    test_raw_630w10a[
        cycle_col_630w10a
    ]
    .to_numpy(
        dtype=np.int64
    )
)


unique_test_engines_630w10a = np.unique(
    test_unit_ids_raw_630w10a
)

if unique_test_engines_630w10a.size != 248:
    raise RuntimeError(
        "FD004 TEST engine count is not 248."
    )

expected_engine_ids_630w10a = np.arange(
    1,
    249,
    dtype=np.int64,
)

if not np.array_equal(
    unique_test_engines_630w10a,
    expected_engine_ids_630w10a,
):
    raise RuntimeError(
        "FD004 TEST engine IDs are not exactly 1..248."
    )

if official_rul_630w10a.shape != (
    248,
):
    raise RuntimeError(
        "FD004 official RUL count is not 248."
    )


if bool(
    test_raw_630w10a.duplicated().any()
):
    raise RuntimeError(
        "Duplicate raw rows detected in FD004 TEST."
    )


for engine_id_630w10a in expected_engine_ids_630w10a:

    engine_cycles_630w10a = (
        test_raw_630w10a.loc[
            test_raw_630w10a[
                unit_col_630w10a
            ].astype(np.int64)
            ==
            int(
                engine_id_630w10a
            ),
            cycle_col_630w10a,
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    expected_cycles_630w10a = np.arange(
        1,
        engine_cycles_630w10a.size + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        engine_cycles_630w10a,
        expected_cycles_630w10a,
    ):
        raise RuntimeError(
            "FD004 TEST cycle ordering failed for engine "
            + str(
                engine_id_630w10a
            )
        )


# =============================================================================
# 7) VERIFY FROZEN REGIME-ASSIGNMENT RULE AGAINST RESIDENT DEV AUTHORITY
#
# Canonical KMeans assignment = nearest frozen canonical center in the
# original operating-setting space.
# We prove that this reproduces ALL resident DEV regime IDs exactly before
# applying it to TEST.
# =============================================================================

canonical_centers_630w10a = np.asarray(
    FD004_CANONICAL_REGIME_CENTERS_630F,
    dtype=np.float64,
)

if canonical_centers_630w10a.shape != (
    6,
    3,
):
    raise RuntimeError(
        "Frozen FD004 canonical regime centers changed shape."
    )


dev_base_630w10a = (
    FD004_DEV_CORRUPTION_BASE_630V17A
)

for col_630w10a in op_cols_630w10a:
    if col_630w10a not in dev_base_630w10a.columns:
        raise RuntimeError(
            "Resident DEV corruption base lacks operating-setting column: "
            + str(
                col_630w10a
            )
        )


dev_ops_630w10a = (
    dev_base_630w10a.loc[
        :,
        list(
            op_cols_630w10a
        ),
    ]
    .to_numpy(
        dtype=np.float64
    )
)

dev_center_dist2_630w10a = np.sum(
    (
        dev_ops_630w10a[
            :,
            None,
            :
        ]
        -
        canonical_centers_630w10a[
            None,
            :,
            :
        ]
    )
    ** 2,
    axis=2,
)

dev_regime_reconstructed_630w10a = np.argmin(
    dev_center_dist2_630w10a,
    axis=1,
).astype(
    np.int64
)

dev_regime_authority_630w10a = np.asarray(
    FD004_DEV_REGIME_IDS_630F,
    dtype=np.int64,
).reshape(-1)

if not np.array_equal(
    dev_regime_reconstructed_630w10a,
    dev_regime_authority_630w10a,
):
    mismatch_count_630w10a = int(
        np.sum(
            dev_regime_reconstructed_630w10a
            !=
            dev_regime_authority_630w10a
        )
    )

    raise RuntimeError(
        "Frozen canonical-center assignment does not reproduce resident "
        f"DEV regimes exactly; mismatches={mismatch_count_630w10a}"
    )

dev_regime_assignment_exact_630w10a = True


# =============================================================================
# 8) ASSIGN TEST REGIMES USING THE NOW-VALIDATED FROZEN RULE
# =============================================================================

test_ops_630w10a = (
    test_raw_630w10a.loc[
        :,
        list(
            op_cols_630w10a
        ),
    ]
    .to_numpy(
        dtype=np.float64
    )
)

test_center_dist2_630w10a = np.sum(
    (
        test_ops_630w10a[
            :,
            None,
            :
        ]
        -
        canonical_centers_630w10a[
            None,
            :,
            :
        ]
    )
    ** 2,
    axis=2,
)

test_regime_ids_630w10a = np.argmin(
    test_center_dist2_630w10a,
    axis=1,
).astype(
    np.int64
)

if test_regime_ids_630w10a.shape != (
    len(
        test_raw_630w10a
    ),
):
    raise RuntimeError(
        "TEST regime-ID vector length mismatch."
    )

if np.any(
    (test_regime_ids_630w10a < 0)
    |
    (test_regime_ids_630w10a > 5)
):
    raise RuntimeError(
        "TEST regime ID escaped expected canonical range 0..5."
    )


test_regime_counts_630w10a = tuple(
    int(x)
    for x in np.bincount(
        test_regime_ids_630w10a,
        minlength=6,
    )
)


# =============================================================================
# 9) FROZEN TEST NORMALIZATION
# =============================================================================

test_normalized_630w10a = (
    normalize_fd004_raw_630v2(
        test_raw_630w10a.copy(),
        test_regime_ids_630w10a.copy(),
    )
)

if not isinstance(
    test_normalized_630w10a,
    pd.DataFrame,
):
    raise RuntimeError(
        "Frozen FD004 normalizer did not return a DataFrame."
    )

numeric_normalized_test_630w10a = (
    test_normalized_630w10a
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    numeric_normalized_test_630w10a
).all():
    raise RuntimeError(
        "Frozen TEST normalization produced non-finite values."
    )


# =============================================================================
# 10) FROZEN 30-CYCLE TEST WINDOW CONSTRUCTION
# =============================================================================

window_result_630w10a = (
    build_windows_fd004_630v2(
        test_normalized_630w10a
    )
)

if not (
    isinstance(
        window_result_630w10a,
        tuple,
    )
    and
    len(
        window_result_630w10a
    )
    ==
    3
):
    raise RuntimeError(
        "Frozen FD004 window builder return contract changed."
    )


(
    X_TEST_FD004_630W10A,
    TEST_WINDOW_UNIT_IDS_FD004_630W10A,
    TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A,
) = window_result_630w10a


X_TEST_FD004_630W10A = np.asarray(
    X_TEST_FD004_630W10A
)

TEST_WINDOW_UNIT_IDS_FD004_630W10A = np.asarray(
    TEST_WINDOW_UNIT_IDS_FD004_630W10A,
    dtype=np.int64,
).reshape(-1)

TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A = np.asarray(
    TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A,
    dtype=np.int64,
).reshape(-1)


if X_TEST_FD004_630W10A.ndim != 3:
    raise RuntimeError(
        "TEST window tensor is not rank-3."
    )

if X_TEST_FD004_630W10A.shape[1:] != (
    30,
    24,
):
    raise RuntimeError(
        "TEST window tensor does not obey frozen (30,24) window rule. "
        f"Observed: {X_TEST_FD004_630W10A.shape}"
    )

if not (
    X_TEST_FD004_630W10A.shape[0]
    ==
    TEST_WINDOW_UNIT_IDS_FD004_630W10A.size
    ==
    TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A.size
):
    raise RuntimeError(
        "TEST window tensor/unit/cycle lengths disagree."
    )

if not np.isfinite(
    X_TEST_FD004_630W10A
).all():
    raise RuntimeError(
        "TEST window tensor contains non-finite values."
    )


expected_window_count_630w10a = int(
    sum(
        max(
            0,
            int(
                np.sum(
                    test_unit_ids_raw_630w10a
                    ==
                    engine_id_630w10a
                )
            )
            -
            30
            +
            1,
        )
        for engine_id_630w10a
        in expected_engine_ids_630w10a
    )
)

if X_TEST_FD004_630W10A.shape[0] != (
    expected_window_count_630w10a
):
    raise RuntimeError(
        "TEST window count does not match frozen stride-1, length-30 rule."
    )


# =============================================================================
# 11) IDENTIFY EXACTLY ONE FINAL WINDOW PER TEST ENGINE
#
# No predictions or errors are involved here.
# =============================================================================

endpoint_window_indices_630w10a = []
endpoint_unit_ids_630w10a = []
endpoint_cycles_630w10a = []


for engine_id_630w10a in expected_engine_ids_630w10a:

    positions_630w10a = np.flatnonzero(
        TEST_WINDOW_UNIT_IDS_FD004_630W10A
        ==
        engine_id_630w10a
    )

    if positions_630w10a.size <= 0:
        raise RuntimeError(
            "No valid 30-cycle TEST window exists for engine "
            + str(
                engine_id_630w10a
            )
        )

    final_position_630w10a = int(
        positions_630w10a[
            -1
        ]
    )

    raw_final_cycle_630w10a = int(
        test_raw_630w10a.loc[
            test_raw_630w10a[
                unit_col_630w10a
            ].astype(np.int64)
            ==
            engine_id_630w10a,
            cycle_col_630w10a,
        ].max()
    )

    window_final_cycle_630w10a = int(
        TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A[
            final_position_630w10a
        ]
    )

    if (
        window_final_cycle_630w10a
        !=
        raw_final_cycle_630w10a
    ):
        raise RuntimeError(
            "Final TEST window does not terminate at raw final cycle "
            "for engine "
            + str(
                engine_id_630w10a
            )
        )

    endpoint_window_indices_630w10a.append(
        final_position_630w10a
    )

    endpoint_unit_ids_630w10a.append(
        int(
            engine_id_630w10a
        )
    )

    endpoint_cycles_630w10a.append(
        window_final_cycle_630w10a
    )


FD004_TEST_ENDPOINT_WINDOW_INDICES_630W10A = np.asarray(
    endpoint_window_indices_630w10a,
    dtype=np.int64,
)

FD004_TEST_ENDPOINT_UNIT_IDS_630W10A = np.asarray(
    endpoint_unit_ids_630w10a,
    dtype=np.int64,
)

FD004_TEST_ENDPOINT_CYCLES_630W10A = np.asarray(
    endpoint_cycles_630w10a,
    dtype=np.int64,
)


if FD004_TEST_ENDPOINT_WINDOW_INDICES_630W10A.shape != (
    248,
):
    raise RuntimeError(
        "TEST endpoint-window count is not 248."
    )

if not np.array_equal(
    FD004_TEST_ENDPOINT_UNIT_IDS_630W10A,
    expected_engine_ids_630w10a,
):
    raise RuntimeError(
        "TEST endpoint engine ordering is not exactly 1..248."
    )


# =============================================================================
# 12) BIND OFFICIAL RUL TO THE FIXED TEST ENGINE ORDER
#
# IMPORTANT:
# We bind labels here but do NOT compare them to any predictions.
# =============================================================================

FD004_TEST_OFFICIAL_RUL_630W10A = (
    official_rul_630w10a.copy()
)

FD004_TEST_ENDPOINT_LABEL_BINDING_630W10A = pd.DataFrame(
    {
        "unit_id":
            FD004_TEST_ENDPOINT_UNIT_IDS_630W10A,

        "endpoint_cycle":
            FD004_TEST_ENDPOINT_CYCLES_630W10A,

        "official_rul":
            FD004_TEST_OFFICIAL_RUL_630W10A,
    }
)

if FD004_TEST_ENDPOINT_LABEL_BINDING_630W10A.shape != (
    248,
    3,
):
    raise RuntimeError(
        "TEST endpoint label binding shape changed."
    )


# =============================================================================
# 13) HASH FROZEN TEST INPUT OBJECTS
# =============================================================================

FD004_TEST_RAW_NUMERIC_SHA256_630W10A = str(
    sha256_array_630v2(
        numeric_test_630w10a
    )
)

FD004_TEST_REGIME_IDS_SHA256_630W10A = str(
    sha256_array_630v2(
        test_regime_ids_630w10a
    )
)

FD004_TEST_WINDOWS_SHA256_630W10A = str(
    sha256_array_630v2(
        X_TEST_FD004_630W10A
    )
)

FD004_TEST_WINDOW_UNIT_IDS_SHA256_630W10A = str(
    sha256_array_630v2(
        TEST_WINDOW_UNIT_IDS_FD004_630W10A
    )
)

FD004_TEST_WINDOW_ENDPOINT_CYCLES_SHA256_630W10A = str(
    sha256_array_630v2(
        TEST_WINDOW_ENDPOINT_CYCLES_FD004_630W10A
    )
)

FD004_TEST_ENDPOINT_INDICES_SHA256_630W10A = str(
    sha256_array_630v2(
        FD004_TEST_ENDPOINT_WINDOW_INDICES_630W10A
    )
)

FD004_TEST_OFFICIAL_RUL_SHA256_630W10A = str(
    sha256_array_630v2(
        FD004_TEST_OFFICIAL_RUL_630W10A
    )
)


# =============================================================================
# 14) FREEZE TEST-INPUT BINDING CONTRACT
# =============================================================================

FD004_TEST_INPUT_BINDING_630W10A = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-10A",

    "status":
        "HELDOUT_TEST_AND_RUL_OPENED_INPUTS_BOUND_NO_INFERENCE",

    "one_way_admission_sha256":
        str(
            FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9
        ),

    "test_path":
        str(
            test_path_630w10a
        ),

    "rul_path":
        str(
            rul_path_630w10a
        ),

    "test_file_sha256":
        FD004_TEST_FILE_SHA256_630W10A,

    "rul_file_sha256":
        FD004_RUL_FILE_SHA256_630W10A,

    "raw_column_schema":
        tuple(
            str(x)
            for x in raw_columns_630w10a
        ),

    "test_raw_shape":
        tuple(
            int(x)
            for x in test_raw_630w10a.shape
        ),

    "test_engine_count":
        int(
            unique_test_engines_630w10a.size
        ),

    "official_rul_count":
        int(
            FD004_TEST_OFFICIAL_RUL_630W10A.size
        ),

    "dev_regime_assignment_rule_reproduced_exactly":
        True,

    "test_regime_counts":
        test_regime_counts_630w10a,

    "test_regime_ids_sha256":
        FD004_TEST_REGIME_IDS_SHA256_630W10A,

    "window_rule":
        {
            "length":
                30,

            "stride":
                1,

            "feature_count":
                24,
        },

    "test_window_shape":
        tuple(
            int(x)
            for x in X_TEST_FD004_630W10A.shape
        ),

    "test_windows_sha256":
        FD004_TEST_WINDOWS_SHA256_630W10A,

    "endpoint_count":
        int(
            FD004_TEST_ENDPOINT_WINDOW_INDICES_630W10A.size
        ),

    "endpoint_indices_sha256":
        FD004_TEST_ENDPOINT_INDICES_SHA256_630W10A,

    "official_rul_sha256":
        FD004_TEST_OFFICIAL_RUL_SHA256_630W10A,

    "test_contents_read":
        True,

    "official_rul_contents_read":
        True,

    "model_inference_performed":
        False,

    "mc_dropout_executed":
        False,

    "test_risk_computed":
        False,

    "test_prediction_errors_computed":
        False,

    "test_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,

    "model_retrained":
        False,

    "preprocessing_refit":
        False,

    "risk_recalibrated":
        False,

    "test_tuning_permitted":
        False,
}


FD004_TEST_INPUT_BINDING_SHA256_630W10A = (
    _json_sha256_630w10a(
        FD004_TEST_INPUT_BINDING_630W10A
    )
)


# =============================================================================
# 15) RESTART-SAFE CHECKPOINT
# =============================================================================

checkpoint_path_630w10a = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_test_input_binding_630w10a_v1.json"
)

checkpoint_dir_630w10a = os.path.dirname(
    checkpoint_path_630w10a
)

if checkpoint_dir_630w10a:
    os.makedirs(
        checkpoint_dir_630w10a,
        exist_ok=True,
    )


checkpoint_payload_630w10a = {
    "FD004_TEST_INPUT_BINDING_630W10A":
        FD004_TEST_INPUT_BINDING_630W10A,

    "FD004_TEST_INPUT_BINDING_SHA256_630W10A":
        FD004_TEST_INPUT_BINDING_SHA256_630W10A,
}


tmp_path_630w10a = (
    checkpoint_path_630w10a
    +
    ".tmp"
)

with open(
    tmp_path_630w10a,
    "w",
    encoding="utf-8",
) as handle_630w10a:
    json.dump(
        checkpoint_payload_630w10a,
        handle_630w10a,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )

os.replace(
    tmp_path_630w10a,
    checkpoint_path_630w10a,
)


with open(
    checkpoint_path_630w10a,
    "rb",
) as handle_630w10a:
    checkpoint_bytes_630w10a = (
        handle_630w10a.read()
    )

FD004_TEST_INPUT_CHECKPOINT_FILE_SHA256_630W10A = (
    hashlib.sha256(
        checkpoint_bytes_630w10a
    ).hexdigest()
)


with open(
    checkpoint_path_630w10a,
    "r",
    encoding="utf-8",
) as handle_630w10a:
    checkpoint_roundtrip_630w10a = json.load(
        handle_630w10a
    )


checkpoint_roundtrip_exact_630w10a = bool(
    checkpoint_roundtrip_630w10a
    ==
    checkpoint_payload_630w10a
)

if not checkpoint_roundtrip_exact_630w10a:
    raise RuntimeError(
        "TEST-input binding checkpoint roundtrip failed."
    )


# =============================================================================
# 16) FINAL FLAGS
# =============================================================================

FD004_TEST_INPUTS_BOUND_630W10A = True

FD004_TEST_MODEL_INFERENCE_PERFORMED_630W10A = False
FD004_TEST_MC_EXECUTED_630W10A = False
FD004_TEST_RISK_COMPUTED_630W10A = False
FD004_TEST_OUTCOMES_COMPUTED_630W10A = False

FD004_THRESHOLD_MODIFIED_AFTER_TEST_ACCESS_630W10A = False
FD004_THRESHOLD_RECOMPUTED_AFTER_TEST_ACCESS_630W10A = False
FD004_POLICY_MODIFIED_AFTER_TEST_ACCESS_630W10A = False
FD004_MODEL_RETRAINED_AFTER_TEST_ACCESS_630W10A = False
FD004_PREPROCESSING_REFIT_AFTER_TEST_ACCESS_630W10A = False
FD004_RISK_RECALIBRATED_AFTER_TEST_ACCESS_630W10A = False

all_checks_passed_630w10a = True


# =============================================================================
# 17) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-10A — FD004 FIRST HELD-OUT "
    "TEST/RUL ACCESS + INPUT BIND"
)

print("=" * 176)

print(
    "W-9 admission SHA exact           :",
    admission_sha_exact_630w10a,
)

print(
    "TEST contents read                :",
    FD004_TEST_CONTENTS_READ_630W10A,
)

print(
    "Official RUL contents read        :",
    FD004_RUL_CONTENTS_READ_630W10A,
)

print(
    "Model inference performed         :",
    False,
)

print(
    "MC dropout executed               :",
    False,
)

print(
    "TEST risk computed                :",
    False,
)

print(
    "TEST prediction errors computed   :",
    False,
)

print(
    "TEST outcome metrics computed     :",
    False,
)


print("\n--- FILE EVIDENCE ---")

print(
    "TEST path                         :",
    test_path_630w10a,
)

print(
    "TEST file SHA-256                 :",
    FD004_TEST_FILE_SHA256_630W10A,
)

print(
    "RUL path                          :",
    rul_path_630w10a,
)

print(
    "RUL file SHA-256                  :",
    FD004_RUL_FILE_SHA256_630W10A,
)


print("\n--- RAW TEST VALIDATION ---")

print(
    "TEST raw shape                    :",
    test_raw_630w10a.shape,
)

print(
    "TEST engine count                 :",
    unique_test_engines_630w10a.size,
)

print(
    "TEST engine IDs exact 1..248      :",
    np.array_equal(
        unique_test_engines_630w10a,
        expected_engine_ids_630w10a,
    ),
)

print(
    "Official RUL count                :",
    FD004_TEST_OFFICIAL_RUL_630W10A.size,
)

print(
    "Raw duplicate rows                :",
    int(
        test_raw_630w10a
        .duplicated()
        .sum()
    ),
)


print("\n--- FROZEN REGIME ASSIGNMENT ---")

print(
    "DEV rule reproduced exactly       :",
    dev_regime_assignment_exact_630w10a,
)

print(
    "TEST regime counts r0..r5         :",
    test_regime_counts_630w10a,
)


print("\n--- FROZEN TEST WINDOWS ---")

print(
    "Window tensor shape               :",
    X_TEST_FD004_630W10A.shape,
)

print(
    "Expected window count             :",
    expected_window_count_630w10a,
)

print(
    "Endpoint count                    :",
    FD004_TEST_ENDPOINT_WINDOW_INDICES_630W10A.size,
)

print(
    "Endpoint engine IDs exact 1..248  :",
    np.array_equal(
        FD004_TEST_ENDPOINT_UNIT_IDS_630W10A,
        expected_engine_ids_630w10a,
    ),
)


print("\n--- HASH EVIDENCE ---")

print(
    "TEST regime-ID SHA-256            :",
    FD004_TEST_REGIME_IDS_SHA256_630W10A,
)

print(
    "TEST windows SHA-256              :",
    FD004_TEST_WINDOWS_SHA256_630W10A,
)

print(
    "Endpoint-index SHA-256            :",
    FD004_TEST_ENDPOINT_INDICES_SHA256_630W10A,
)

print(
    "Official RUL SHA-256              :",
    FD004_TEST_OFFICIAL_RUL_SHA256_630W10A,
)

print(
    "TEST-input binding SHA-256        :",
    FD004_TEST_INPUT_BINDING_SHA256_630W10A,
)

print(
    "Checkpoint path                   :",
    checkpoint_path_630w10a,
)

print(
    "Checkpoint file SHA-256           :",
    FD004_TEST_INPUT_CHECKPOINT_FILE_SHA256_630W10A,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_630w10a,
)


print("\n--- IMMUTABILITY AFTER TEST ACCESS ---")

print("Threshold modified                :", False)
print("Threshold recomputed              :", False)
print("Policy modified                   :", False)
print("Model retrained                   :", False)
print("Preprocessing refit               :", False)
print("Risk recalibrated                 :", False)
print("TEST tuning permitted             :", False)


print(
    "\nPASS — FD004 held-out TEST and official RUL have now been "
    "opened for the first time and bound under the frozen protocol."
)

print(
    "No model inference, uncertainty/risk computation, prediction "
    "error, or TEST outcome metric was performed in this cell."
)

print(
    "From this point onward TEST/RUL are permanently considered opened; "
    "no tuning or methodological changes are permitted."
)

print(
    "Next: run frozen deterministic + MC inference and frozen risk "
    "construction on TEST, then extract the 248 endpoint predictions/risk "
    "under the unchanged policy before computing final held-out metrics."
)

In [ ]:
# STEP 6.30W-10A-R1 — POST-ACCESS SHORT-TEST-TRAJECTORY METHODOLOGY RECOVERY
#
# TEST/RUL WERE ALREADY OPENED IN THE FAILED W-10A CELL.
#
# This recovery cell:
# - records that fact truthfully
# - audits which FD004 TEST engines have <30 observed rows
# - DOES NOT create windows
# - DOES NOT pad/truncate/impute
# - DOES NOT run model inference
# - DOES NOT run MC dropout
# - DOES NOT compute risk/errors/MAE
# - statically searches resident historical notebook source for the
#   prior FD002/FD003 held-out TEST endpoint handling rule
#
# We will not choose a short-sequence handling rule until historical
# evidence is recovered.

import ast
import hashlib
import json
import re
import numpy as np
import pandas as pd


# =============================================================================
# 0) POST-FAILURE STATE GUARDS
# =============================================================================

required_630w10ar1 = (
    "all_checks_passed_630w9",
    "FD004_TEST_CONTENTS_READ_630W10A",
    "FD004_RUL_CONTENTS_READ_630W10A",
    "test_raw_630w10a",
    "official_rul_630w10a",
)

missing_630w10ar1 = [
    name
    for name in required_630w10ar1
    if name not in globals()
]

if missing_630w10ar1:
    raise RuntimeError(
        "Missing post-W-10A recovery object(s): "
        + ", ".join(missing_630w10ar1)
    )

if not bool(all_checks_passed_630w9):
    raise RuntimeError(
        "W-9 admission must remain passed."
    )

if not bool(
    FD004_TEST_CONTENTS_READ_630W10A
):
    raise RuntimeError(
        "Recovery expected TEST to already be marked opened."
    )

if not bool(
    FD004_RUL_CONTENTS_READ_630W10A
):
    raise RuntimeError(
        "Recovery expected official RUL to already be marked opened."
    )


# =============================================================================
# 1) AUDIT FD004 TEST TRAJECTORY LENGTHS ONLY
# =============================================================================

test_frame_630w10ar1 = (
    test_raw_630w10a
    .copy()
    .reset_index(drop=True)
)

if "unit_id" in test_frame_630w10ar1.columns:
    unit_col_630w10ar1 = "unit_id"
elif "unit" in test_frame_630w10ar1.columns:
    unit_col_630w10ar1 = "unit"
else:
    unit_col_630w10ar1 = test_frame_630w10ar1.columns[0]


test_lengths_630w10ar1 = (
    test_frame_630w10ar1
    .groupby(
        unit_col_630w10ar1,
        sort=True,
    )
    .size()
    .astype(np.int64)
)

if len(
    test_lengths_630w10ar1
) != 248:
    raise RuntimeError(
        "FD004 TEST engine count is no longer 248."
    )

short_test_lengths_630w10ar1 = (
    test_lengths_630w10ar1.loc[
        test_lengths_630w10ar1
        <
        30
    ]
)

short_engine_count_630w10ar1 = int(
    short_test_lengths_630w10ar1.shape[0]
)

minimum_test_length_630w10ar1 = int(
    test_lengths_630w10ar1.min()
)

maximum_test_length_630w10ar1 = int(
    test_lengths_630w10ar1.max()
)


# =============================================================================
# 2) BUILD STATIC SOURCE CORPUS
# =============================================================================

source_corpus_630w10ar1 = []
seen_sha_630w10ar1 = set()


def _add_source_630w10ar1(
    origin,
    sequence,
    text,
):
    if not isinstance(
        text,
        str,
    ):
        return

    if len(text) < 80:
        return

    sha = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if sha in seen_sha_630w10ar1:
        return

    seen_sha_630w10ar1.add(
        sha
    )

    source_corpus_630w10ar1.append(
        {
            "origin":
                origin,

            "sequence":
                sequence,

            "sha256":
                sha,

            "text":
                text,
        }
    )


if "In" in globals():
    try:
        for idx_630w10ar1, text_630w10ar1 in enumerate(
            In
        ):
            _add_source_630w10ar1(
                "In",
                idx_630w10ar1,
                text_630w10ar1,
            )
    except Exception:
        pass


for name_630w10ar1, value_630w10ar1 in list(
    globals().items()
):
    if not isinstance(
        value_630w10ar1,
        str,
    ):
        continue

    upper_name_630w10ar1 = (
        name_630w10ar1.upper()
    )

    if any(
        token_630w10ar1
        in
        upper_name_630w10ar1
        for token_630w10ar1
        in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "PROTOCOL",
            "CHECKPOINT",
            "TEST",
        )
    ):
        _add_source_630w10ar1(
            "global:" + name_630w10ar1,
            None,
            value_630w10ar1,
        )


# =============================================================================
# 3) TARGET HISTORICAL HELD-OUT TEST HANDLING SIGNALS
# =============================================================================

strong_signals_630w10ar1 = (
    "FD003",
    "FD002",
    "TEST",
    "endpoint",
    "window",
)

handling_signals_630w10ar1 = (
    "padding",
    "pad",
    "left_pad",
    "repeat",
    "tile",
    "fewer than 30",
    "< 30",
    "window_length",
    "sequence_length",
    "last 30",
    "final 30",
    "endpoint_window",
    "endpoint_indices",
    "endpoint_index",
    "final_window",
    "test_windows",
    "X_TEST",
    "official_rul",
    "RUL_FD003",
    "RUL_FD002",
    "build_windows",
)


candidate_sources_630w10ar1 = []

for record_630w10ar1 in source_corpus_630w10ar1:

    text_630w10ar1 = (
        record_630w10ar1[
            "text"
        ]
    )

    lower_630w10ar1 = (
        text_630w10ar1.lower()
    )

    has_prior_dataset_630w10ar1 = (
        "fd003"
        in lower_630w10ar1
        or
        "fd002"
        in lower_630w10ar1
    )

    has_test_630w10ar1 = (
        "test"
        in lower_630w10ar1
    )

    if not (
        has_prior_dataset_630w10ar1
        and
        has_test_630w10ar1
    ):
        continue


    matched_signals_630w10ar1 = tuple(
        signal_630w10ar1
        for signal_630w10ar1
        in handling_signals_630w10ar1
        if signal_630w10ar1.lower()
        in lower_630w10ar1
    )

    if not matched_signals_630w10ar1:
        continue


    score_630w10ar1 = (
        5
        *
        len(
            matched_signals_630w10ar1
        )
    )

    if (
        "rul_fd003"
        in lower_630w10ar1
        or
        "rul_fd002"
        in lower_630w10ar1
    ):
        score_630w10ar1 += 20

    if (
        "endpoint"
        in lower_630w10ar1
        and
        "window"
        in lower_630w10ar1
    ):
        score_630w10ar1 += 20

    if (
        "last 30"
        in lower_630w10ar1
        or
        "final 30"
        in lower_630w10ar1
    ):
        score_630w10ar1 += 20

    if (
        "pad"
        in lower_630w10ar1
        or
        "padding"
        in lower_630w10ar1
    ):
        score_630w10ar1 += 20

    if (
        "accepted_mae"
        in lower_630w10ar1
        or
        "test_full_mae"
        in lower_630w10ar1
    ):
        score_630w10ar1 += 10


    candidate_sources_630w10ar1.append(
        {
            "origin":
                record_630w10ar1[
                    "origin"
                ],

            "sequence":
                record_630w10ar1[
                    "sequence"
                ],

            "sha256":
                record_630w10ar1[
                    "sha256"
                ],

            "score":
                int(
                    score_630w10ar1
                ),

            "matched_signals":
                matched_signals_630w10ar1,

            "text":
                text_630w10ar1,
        }
    )


candidate_sources_630w10ar1 = sorted(
    candidate_sources_630w10ar1,
    key=lambda row: (
        -row[
            "score"
        ],
        (
            -1
            if row[
                "sequence"
            ]
            is None
            else
            row[
                "sequence"
            ]
        ),
    ),
)


# =============================================================================
# 4) EXTRACT ONLY RELEVANT SOURCE CONTEXT
# =============================================================================

context_regex_630w10ar1 = re.compile(
    r"FD003|FD002|"
    r"RUL_FD003|RUL_FD002|"
    r"X_TEST|"
    r"endpoint|"
    r"final[_ ]?window|"
    r"last\s*30|"
    r"final\s*30|"
    r"fewer\s+than\s+30|"
    r"pad|padding|"
    r"build_windows|"
    r"window_length|"
    r"sequence_length",
    flags=re.IGNORECASE,
)


def _extract_contexts_630w10ar1(
    text,
):
    lines_630w10ar1 = (
        text.splitlines()
    )

    matches_630w10ar1 = [
        idx_630w10ar1
        for idx_630w10ar1, line_630w10ar1
        in enumerate(
            lines_630w10ar1,
            start=1,
        )
        if context_regex_630w10ar1.search(
            line_630w10ar1
        )
    ]

    contexts_630w10ar1 = []

    for line_no_630w10ar1 in matches_630w10ar1[
        :40
    ]:

        start_630w10ar1 = max(
            1,
            line_no_630w10ar1 - 8,
        )

        end_630w10ar1 = min(
            len(
                lines_630w10ar1
            ),
            line_no_630w10ar1 + 12,
        )

        contexts_630w10ar1.append(
            {
                "match_line":
                    line_no_630w10ar1,

                "snippet":
                    tuple(
                        (
                            idx_630w10ar1,
                            lines_630w10ar1[
                                idx_630w10ar1 - 1
                            ],
                        )
                        for idx_630w10ar1
                        in range(
                            start_630w10ar1,
                            end_630w10ar1 + 1,
                        )
                    ),
            }
        )

    return contexts_630w10ar1


top_sources_630w10ar1 = []

for row_630w10ar1 in candidate_sources_630w10ar1[
    :12
]:
    top_sources_630w10ar1.append(
        {
            "origin":
                row_630w10ar1[
                    "origin"
                ],

            "sequence":
                row_630w10ar1[
                    "sequence"
                ],

            "sha256":
                row_630w10ar1[
                    "sha256"
                ],

            "score":
                row_630w10ar1[
                    "score"
                ],

            "matched_signals":
                row_630w10ar1[
                    "matched_signals"
                ],

            "contexts":
                _extract_contexts_630w10ar1(
                    row_630w10ar1[
                        "text"
                    ]
                ),
        }
    )


# =============================================================================
# 5) FREEZE RECOVERY AUDIT
# =============================================================================

audit_payload_630w10ar1 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-10A-R1",

    "status":
        "POST_TEST_ACCESS_SHORT_TRAJECTORY_METHOD_RECOVERY_ONLY",

    "test_contents_already_read":
        True,

    "official_rul_contents_already_read":
        True,

    "test_engine_count":
        int(
            test_lengths_630w10ar1.size
        ),

    "minimum_test_trajectory_length":
        minimum_test_length_630w10ar1,

    "maximum_test_trajectory_length":
        maximum_test_length_630w10ar1,

    "shorter_than_30_engine_count":
        short_engine_count_630w10ar1,

    "shorter_than_30_engine_lengths":
        tuple(
            (
                int(engine_id_630w10ar1),
                int(length_630w10ar1),
            )
            for engine_id_630w10ar1, length_630w10ar1
            in short_test_lengths_630w10ar1.items()
        ),

    "historical_candidate_source_count":
        len(
            candidate_sources_630w10ar1
        ),

    "model_inference_performed":
        False,

    "mc_dropout_executed":
        False,

    "test_risk_computed":
        False,

    "test_prediction_errors_computed":
        False,

    "test_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,

    "preprocessing_refit":
        False,

    "risk_recalibrated":
        False,
}


FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1 = (
    audit_payload_630w10ar1
)

FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630w10ar1,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

FD004_HISTORICAL_TEST_HANDLING_SOURCE_CANDIDATES_630W10AR1 = (
    top_sources_630w10ar1
)

all_checks_passed_630w10ar1 = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-10A-R1 — POST-ACCESS SHORT-TEST-TRAJECTORY "
    "METHODOLOGY RECOVERY"
)

print("=" * 176)

print("TEST already opened               :", True)
print("Official RUL already opened       :", True)
print("Model inference performed         :", False)
print("MC dropout executed               :", False)
print("TEST risk computed                :", False)
print("TEST errors computed              :", False)
print("TEST outcome metrics computed     :", False)
print("Threshold modified                :", False)
print("Policy modified                   :", False)


print("\n--- FD004 TEST TRAJECTORY LENGTH AUDIT ---")

print(
    "Engine count                      :",
    test_lengths_630w10ar1.size,
)

print(
    "Trajectory length min / max       :",
    (
        minimum_test_length_630w10ar1,
        maximum_test_length_630w10ar1,
    ),
)

print(
    "Engines with <30 observed rows    :",
    short_engine_count_630w10ar1,
)

print(
    "Exact short engine lengths        :",
    tuple(
        (
            int(engine_id_630w10ar1),
            int(length_630w10ar1),
        )
        for engine_id_630w10ar1, length_630w10ar1
        in short_test_lengths_630w10ar1.items()
    ),
)


print("\n--- HISTORICAL TEST-HANDLING SOURCE CANDIDATES ---")

print(
    "Candidate count                   :",
    len(
        candidate_sources_630w10ar1
    ),
)

print(
    "Showing top sources               :",
    len(
        top_sources_630w10ar1
    ),
)


for idx_630w10ar1, row_630w10ar1 in enumerate(
    top_sources_630w10ar1,
    start=1,
):

    print(
        f"\nSOURCE {idx_630w10ar1}"
    )

    print(
        "  origin                          :",
        row_630w10ar1[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        row_630w10ar1[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        row_630w10ar1[
            "sha256"
        ],
    )

    print(
        "  score                           :",
        row_630w10ar1[
            "score"
        ],
    )

    print(
        "  matched signals                 :",
        row_630w10ar1[
            "matched_signals"
        ],
    )

    for context_630w10ar1 in row_630w10ar1[
        "contexts"
    ][
        :16
    ]:

        print(
            f"\n    CONTEXT AROUND LINE "
            f"{context_630w10ar1['match_line']}"
        )

        for (
            line_no_630w10ar1,
            line_630w10ar1,
        ) in context_630w10ar1[
            "snippet"
        ]:

            print(
                f"      "
                f"{line_no_630w10ar1:05d} | "
                f"{line_630w10ar1}"
            )


print("\n--- AUDIT ---")

print(
    "Recovery audit SHA-256            :",
    FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1,
)

print(
    "\nPASS — the failed W-10A cell is now correctly classified:"
)

print(
    "TEST/RUL were opened and structurally inspected, but no model "
    "inference, uncertainty/risk, prediction error, or outcome metric "
    "was computed."
)

print(
    "No short-trajectory handling rule has been invented or applied."
)

print(
    "Next: recover the exact historical held-out TEST sequence/window "
    "handling rule, then resume fixed one-way TEST evaluation without tuning."
)

In [ ]:
# STEP 6.30W-10A-R2 — TARGETED HISTORICAL TEST ENDPOINT RULE RECOVERY
#
# TEST/RUL ARE ALREADY OPEN.
#
# STATIC / RESIDENT-EVIDENCE RECOVERY ONLY.
#
# NO TEST window construction.
# NO padding/truncation/repetition applied.
# NO model inference.
# NO MC dropout.
# NO risk computation.
# NO TEST error/MAE.
# NO threshold/policy/model/preprocessing change.
#
# Goal:
# Recover the ACTUAL FD002/FD003 held-out TEST endpoint sequence rule,
# especially what happened when an observed TEST trajectory had <30 cycles.

import inspect
import json
import hashlib
import re
import numbers
import numpy as np


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w10ar2 = (
    "all_checks_passed_630w10ar1",
    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1",
    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1",
)

missing_630w10ar2 = [
    name
    for name in required_630w10ar2
    if name not in globals()
]

if missing_630w10ar2:
    raise RuntimeError(
        "Missing W-10A-R2 authority object(s): "
        + ", ".join(missing_630w10ar2)
    )

if not bool(all_checks_passed_630w10ar1):
    raise RuntimeError(
        "W-10A-R1 must remain passed."
    )


r1_sha_recomputed_630w10ar2 = hashlib.sha256(
    json.dumps(
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

if (
    r1_sha_recomputed_630w10ar2
    !=
    str(
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1
    )
):
    raise RuntimeError(
        "W-10A-R1 audit object changed."
    )


# =============================================================================
# 1) SEARCH PRIOR FD002 / FD003 CONTRACT OBJECTS RECURSIVELY
# =============================================================================

contract_hits_630w10ar2 = []

contract_signal_re_630w10ar2 = re.compile(
    r"test|rul|window|sequence|endpoint|"
    r"short|pad|padding|repeat|tile|"
    r"truncate|length|cycle",
    flags=re.IGNORECASE,
)


def _scalar_text_630w10ar2(value):
    if isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        return repr(value)

    return None


def _walk_contract_630w10ar2(
    root_name,
    value,
    path=(),
    depth=0,
):
    if depth > 7:
        return

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():

            key_text = str(
                key
            )

            new_path = (
                path
                +
                (
                    key_text,
                )
            )

            child_scalar = _scalar_text_630w10ar2(
                child
            )

            searchable = (
                ".".join(
                    new_path
                )
                +
                " "
                +
                (
                    ""
                    if child_scalar is None
                    else child_scalar
                )
            )

            if contract_signal_re_630w10ar2.search(
                searchable
            ):
                if child_scalar is not None:
                    contract_hits_630w10ar2.append(
                        {
                            "root":
                                root_name,

                            "path":
                                ".".join(
                                    new_path
                                ),

                            "value":
                                child_scalar,
                        }
                    )

            _walk_contract_630w10ar2(
                root_name,
                child,
                new_path,
                depth + 1,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) > 200:
            return

        for idx, child in enumerate(
            value
        ):
            _walk_contract_630w10ar2(
                root_name,
                child,
                (
                    path
                    +
                    (
                        f"[{idx}]",
                    )
                ),
                depth + 1,
            )


for name_630w10ar2, value_630w10ar2 in list(
    globals().items()
):

    upper_630w10ar2 = (
        name_630w10ar2.upper()
    )

    if not (
        "FD002"
        in upper_630w10ar2
        or
        "FD003"
        in upper_630w10ar2
    ):
        continue

    if not isinstance(
        value_630w10ar2,
        (
            dict,
            list,
            tuple,
        ),
    ):
        continue

    _walk_contract_630w10ar2(
        name_630w10ar2,
        value_630w10ar2,
    )


# Prefer objects that look like final TEST/reporting/closure evidence.
def _contract_score_630w10ar2(row):

    text = (
        row[
            "root"
        ]
        +
        " "
        +
        row[
            "path"
        ]
        +
        " "
        +
        row[
            "value"
        ]
    ).lower()

    score = 0

    for token, weight in (
        ("test", 20),
        ("endpoint", 20),
        ("window", 20),
        ("sequence", 15),
        ("padding", 30),
        ("pad", 20),
        ("repeat", 20),
        ("truncate", 20),
        ("short", 20),
        ("rul", 10),
        ("closure", 10),
        ("report", 10),
    ):
        if token in text:
            score += weight

    return score


for row_630w10ar2 in contract_hits_630w10ar2:
    row_630w10ar2[
        "score"
    ] = _contract_score_630w10ar2(
        row_630w10ar2
    )


contract_hits_630w10ar2 = sorted(
    contract_hits_630w10ar2,
    key=lambda row: (
        -row[
            "score"
        ],
        row[
            "root"
        ],
        row[
            "path"
        ],
    ),
)


# =============================================================================
# 2) SEARCH RESIDENT FD002 / FD003 CALLABLES FOR TEST/WINDOW LOGIC
# =============================================================================

callable_hits_630w10ar2 = []

direct_code_re_630w10ar2 = re.compile(
    r"np\.pad|"
    r"np\.repeat|"
    r"np\.tile|"
    r"np\.concatenate|"
    r"np\.vstack|"
    r"zeros\s*\(|"
    r"ones\s*\(|"
    r"\[-\s*30\s*:\]|"
    r"\[-\s*window|"
    r"<\s*30|"
    r"fewer\s+than\s+30|"
    r"pad_width|"
    r"padding|"
    r"endpoint|"
    r"final[_ ]?window",
    flags=re.IGNORECASE,
)


for name_630w10ar2, value_630w10ar2 in list(
    globals().items()
):

    upper_630w10ar2 = (
        name_630w10ar2.upper()
    )

    if not (
        "FD002"
        in upper_630w10ar2
        or
        "FD003"
        in upper_630w10ar2
    ):
        continue

    if not callable(
        value_630w10ar2
    ):
        continue

    try:
        source_630w10ar2 = inspect.getsource(
            value_630w10ar2
        )
    except Exception:
        continue

    if not direct_code_re_630w10ar2.search(
        source_630w10ar2
    ):
        continue

    callable_hits_630w10ar2.append(
        {
            "name":
                name_630w10ar2,

            "sha256":
                hashlib.sha256(
                    source_630w10ar2.encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "source":
                source_630w10ar2,
        }
    )


# =============================================================================
# 3) SEARCH HISTORICAL SOURCE TEXT FOR CONCRETE EXECUTABLE TEST LOGIC
#
# Exclude our recent W-series recovery/locator cells.
# =============================================================================

source_records_630w10ar2 = []
seen_source_sha_630w10ar2 = set()


def _add_source_630w10ar2(
    origin,
    sequence,
    text,
):
    if not isinstance(
        text,
        str,
    ):
        return

    if len(text) < 100:
        return

    # Exclude current/recent FD004 W-stage locators.
    lower = text.lower()

    if (
        "step 6.30w-"
        in lower
        or
        "630w10a"
        in lower
    ):
        return

    sha = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if sha in seen_source_sha_630w10ar2:
        return

    seen_source_sha_630w10ar2.add(
        sha
    )

    source_records_630w10ar2.append(
        {
            "origin":
                origin,

            "sequence":
                sequence,

            "sha256":
                sha,

            "text":
                text,
        }
    )


if "In" in globals():
    try:
        for idx, text in enumerate(
            In
        ):
            _add_source_630w10ar2(
                "In",
                idx,
                text,
            )
    except Exception:
        pass


for name_630w10ar2, value_630w10ar2 in list(
    globals().items()
):

    if not isinstance(
        value_630w10ar2,
        str,
    ):
        continue

    upper_630w10ar2 = (
        name_630w10ar2.upper()
    )

    if not any(
        token
        in upper_630w10ar2
        for token in (
            "SOURCE",
            "CELL",
            "HISTORICAL",
            "EXCERPT",
            "PROTOCOL",
        )
    ):
        continue

    _add_source_630w10ar2(
        "global:" + name_630w10ar2,
        None,
        value_630w10ar2,
    )


historical_source_hits_630w10ar2 = []


def _source_score_630w10ar2(
    text
):
    lower = text.lower()

    if not (
        "fd002"
        in lower
        or
        "fd003"
        in lower
    ):
        return -1

    score = 0

    weighted_tokens = (
        ("test_fd003", 60),
        ("test_fd002", 60),
        ("rul_fd003", 50),
        ("rul_fd002", 50),
        ("x_test_fd003", 70),
        ("x_test_fd002", 70),
        ("test_predictions", 40),
        ("test_prediction", 40),
        ("endpoint_window", 50),
        ("endpoint_indices", 40),
        ("final_window", 40),
        ("np.pad", 80),
        ("np.repeat", 80),
        ("np.tile", 70),
        ("np.concatenate", 60),
        ("np.vstack", 40),
        ("pad_width", 80),
        ("padding", 60),
        ("fewer than 30", 60),
        ("< 30", 40),
        ("last 30", 50),
        ("final 30", 50),
        ("[-30:", 60),
        ("window_length", 15),
        ("official_rul", 20),
        ("accepted_mae", 20),
        ("abstained_mae", 20),
    )

    for token, weight in weighted_tokens:
        if token in lower:
            score += weight

    return score


for record_630w10ar2 in source_records_630w10ar2:

    score_630w10ar2 = _source_score_630w10ar2(
        record_630w10ar2[
            "text"
        ]
    )

    if score_630w10ar2 <= 0:
        continue

    historical_source_hits_630w10ar2.append(
        {
            **record_630w10ar2,
            "score":
                score_630w10ar2,
        }
    )


historical_source_hits_630w10ar2 = sorted(
    historical_source_hits_630w10ar2,
    key=lambda row: (
        -row[
            "score"
        ],
        (
            -1
            if row[
                "sequence"
            ]
            is None
            else row[
                "sequence"
            ]
        ),
    ),
)


# =============================================================================
# 4) EXTRACT HIGH-VALUE EXECUTABLE CONTEXTS ONLY
# =============================================================================

executable_line_re_630w10ar2 = re.compile(
    r"np\.pad|"
    r"np\.repeat|"
    r"np\.tile|"
    r"np\.concatenate|"
    r"np\.vstack|"
    r"pad_width|"
    r"padding|"
    r"fewer\s+than\s+30|"
    r"<\s*30|"
    r"\[-\s*30\s*:\]|"
    r"last\s*30|"
    r"final\s*30|"
    r"endpoint_window|"
    r"endpoint_indices|"
    r"final_window|"
    r"X_TEST_FD00[23]|"
    r"test_fd00[23]|"
    r"RUL_FD00[23]",
    flags=re.IGNORECASE,
)


def _compact_contexts_630w10ar2(
    text,
):
    lines = text.splitlines()

    match_lines = [
        idx
        for idx, line
        in enumerate(
            lines,
            start=1,
        )
        if executable_line_re_630w10ar2.search(
            line
        )
    ]

    contexts = []

    used_ranges = []

    for match_line in match_lines:

        start = max(
            1,
            match_line - 6,
        )

        end = min(
            len(lines),
            match_line + 10,
        )

        # Avoid printing overlapping copies of the same source block.
        if any(
            not (
                end < used_start
                or
                start > used_end
            )
            for used_start, used_end
            in used_ranges
        ):
            continue

        used_ranges.append(
            (
                start,
                end,
            )
        )

        contexts.append(
            {
                "match_line":
                    match_line,

                "snippet":
                    tuple(
                        (
                            line_no,
                            lines[
                                line_no - 1
                            ],
                        )
                        for line_no
                        in range(
                            start,
                            end + 1,
                        )
                    ),
            }
        )

        if len(
            contexts
        ) >= 8:
            break

    return contexts


top_source_evidence_630w10ar2 = []

for row_630w10ar2 in historical_source_hits_630w10ar2[
    :12
]:

    contexts_630w10ar2 = _compact_contexts_630w10ar2(
        row_630w10ar2[
            "text"
        ]
    )

    if not contexts_630w10ar2:
        continue

    top_source_evidence_630w10ar2.append(
        {
            "origin":
                row_630w10ar2[
                    "origin"
                ],

            "sequence":
                row_630w10ar2[
                    "sequence"
                ],

            "sha256":
                row_630w10ar2[
                    "sha256"
                ],

            "score":
                row_630w10ar2[
                    "score"
                ],

            "contexts":
                contexts_630w10ar2,
        }
    )


# =============================================================================
# 5) FREEZE RECOVERY EVIDENCE
# =============================================================================

FD004_HISTORICAL_TEST_CONTRACT_HITS_630W10AR2 = tuple(
    contract_hits_630w10ar2[
        :60
    ]
)

FD004_HISTORICAL_TEST_CALLABLE_HITS_630W10AR2 = tuple(
    {
        "name":
            row[
                "name"
            ],

        "sha256":
            row[
                "sha256"
            ],

        "source":
            row[
                "source"
            ],
    }
    for row
    in callable_hits_630w10ar2[
        :20
    ]
)

FD004_HISTORICAL_TEST_SOURCE_EVIDENCE_630W10AR2 = tuple(
    top_source_evidence_630w10ar2
)


audit_payload_630w10ar2 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-10A-R2",

    "status":
        "TARGETED_PRIOR_TEST_ENDPOINT_RULE_RECOVERY_STATIC_ONLY",

    "short_engine_count":
        int(
            FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
                "shorter_than_30_engine_count"
            ]
        ),

    "contract_hit_count":
        len(
            contract_hits_630w10ar2
        ),

    "callable_hit_count":
        len(
            callable_hits_630w10ar2
        ),

    "historical_source_hit_count":
        len(
            historical_source_hits_630w10ar2
        ),

    "display_source_evidence_count":
        len(
            top_source_evidence_630w10ar2
        ),

    "test_contents_already_read":
        True,

    "official_rul_contents_already_read":
        True,

    "test_windows_constructed":
        False,

    "short_sequence_rule_applied":
        False,

    "model_inference_performed":
        False,

    "mc_dropout_executed":
        False,

    "test_risk_computed":
        False,

    "test_errors_computed":
        False,

    "test_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,
}


FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_630W10AR2 = (
    audit_payload_630w10ar2
)

FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2 = (
    hashlib.sha256(
        json.dumps(
            audit_payload_630w10ar2,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()
)

all_checks_passed_630w10ar2 = True


# =============================================================================
# 6) COMPACT REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-10A-R2 — TARGETED HISTORICAL "
    "TEST ENDPOINT RULE RECOVERY"
)

print("=" * 176)

print("TEST/RUL already opened           :", True)
print("TEST windows constructed          :", False)
print("Short-sequence rule applied       :", False)
print("Model inference performed         :", False)
print("MC dropout executed               :", False)
print("TEST risk/errors/metrics computed :", False)
print("Threshold/policy modified         :", False)


print("\n--- R1 SHORT-SEQUENCE FACT ---")

print(
    "FD004 TEST engines <30 cycles     :",
    FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
        "shorter_than_30_engine_count"
    ],
)

print(
    "Exact engine lengths              :",
    FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
        "shorter_than_30_engine_lengths"
    ],
)


print("\n--- PRIOR CONTRACT EVIDENCE ---")

print(
    "Relevant scalar contract hits     :",
    len(
        contract_hits_630w10ar2
    ),
)

for idx, row in enumerate(
    contract_hits_630w10ar2[
        :30
    ],
    start=1,
):
    print(
        f"{idx:02d}. "
        f"{row['root']} :: "
        f"{row['path']} = "
        f"{row['value']}"
    )


print("\n--- PRIOR RESIDENT CALLABLES ---")

print(
    "Callable source hits              :",
    len(
        callable_hits_630w10ar2
    ),
)

for idx, row in enumerate(
    callable_hits_630w10ar2[
        :10
    ],
    start=1,
):
    print(
        f"\nCALLABLE {idx}: "
        f"{row['name']}"
    )

    print(
        "SHA-256                         :",
        row[
            "sha256"
        ],
    )

    print(
        row[
            "source"
        ]
    )


print("\n--- CONCRETE HISTORICAL SOURCE EVIDENCE ---")

print(
    "Historical source hits            :",
    len(
        historical_source_hits_630w10ar2
    ),
)

print(
    "Concrete sources displayed        :",
    len(
        top_source_evidence_630w10ar2
    ),
)


for idx, row in enumerate(
    top_source_evidence_630w10ar2,
    start=1,
):

    print(
        f"\nSOURCE {idx}"
    )

    print(
        "  origin                          :",
        row[
            "origin"
        ],
    )

    print(
        "  sequence                        :",
        row[
            "sequence"
        ],
    )

    print(
        "  SHA-256                         :",
        row[
            "sha256"
        ],
    )

    print(
        "  score                           :",
        row[
            "score"
        ],
    )

    for context in row[
        "contexts"
    ]:

        print(
            f"\n    CONTEXT AROUND LINE "
            f"{context['match_line']}"
        )

        for (
            line_no,
            line,
        ) in context[
            "snippet"
        ]:
            print(
                f"      "
                f"{line_no:05d} | "
                f"{line}"
            )


print("\n--- AUDIT ---")

print(
    "Targeted recovery SHA-256         :",
    FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2,
)

print(
    "\nPASS — only prior resident methodology/source evidence was inspected."
)

print(
    "No FD004 short-sequence handling rule was applied and no TEST "
    "prediction/risk/error/metric was computed."
)

print(
    "Next: bind the exact historical TEST endpoint rule only if this "
    "evidence proves it; otherwise continue source recovery without "
    "inventing a post-TEST method."
)

In [ ]:
# STEP 6.30W-10A-R3 — FD004 PRIMARY HELD-OUT TEST STRUCTURAL-ADMISSIBILITY CLOSURE
#
# The exact historical TEST rule has now been recovered:
# - window length = 30
# - one terminal 30-cycle window per TEST engine
# - if ANY TEST engine has fewer than 30 observed cycles,
#   STOP BEFORE INFERENCE
#
# FD004 has 11 TEST engines shorter than 30 cycles.
#
# Therefore the prospectively frozen PRIMARY held-out TEST evaluation is
# structurally inadmissible under the inherited protocol.
#
# This cell:
# - verifies the resident historical FD003 rule directly
# - verifies the frozen FD004 short-engine evidence
# - freezes the structural non-admissibility decision
# - records that inference was correctly stopped before any TEST prediction
#
# It DOES NOT:
# - pad sequences
# - repeat observations
# - discard short engines
# - change window length
# - use variable-length sequences
# - run deterministic inference
# - run MC dropout
# - compute TEST risk/errors/metrics
# - modify threshold/policy/model/preprocessing/calibration

import os
import json
import hashlib


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w10ar3 = (
    "all_checks_passed_630w10ar2",

    "FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_630W10AR2",
    "FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2",

    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1",
    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1",

    "FD003_TEST_ADMISSION_CONTRACT_629AY",

    "FD004_TEST_CONTENTS_READ_630W10A",
    "FD004_RUL_CONTENTS_READ_630W10A",
)

missing_630w10ar3 = [
    name
    for name in required_630w10ar3
    if name not in globals()
]

if missing_630w10ar3:
    raise RuntimeError(
        "Missing STEP 6.30W-10A-R3 authority object(s): "
        + ", ".join(missing_630w10ar3)
    )

if not bool(
    all_checks_passed_630w10ar2
):
    raise RuntimeError(
        "STEP 6.30W-10A-R2 must remain passed."
    )


# =============================================================================
# 1) VERIFY R1 / R2 RECOVERY EVIDENCE
# =============================================================================

def _json_sha256_630w10ar3(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


r1_sha_exact_630w10ar3 = bool(
    _json_sha256_630w10ar3(
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1
    )
    ==
    str(
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1
    )
)

if not r1_sha_exact_630w10ar3:
    raise RuntimeError(
        "Frozen R1 short-trajectory audit changed."
    )


r2_sha_exact_630w10ar3 = bool(
    _json_sha256_630w10ar3(
        FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_630W10AR2
    )
    ==
    str(
        FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2
    )
)

if not r2_sha_exact_630w10ar3:
    raise RuntimeError(
        "Frozen R2 targeted recovery audit changed."
    )


# =============================================================================
# 2) READ GOVERNING HISTORICAL RULE FROM RESIDENT FD003 CONTRACT
# =============================================================================

historical_contract_630w10ar3 = (
    FD003_TEST_ADMISSION_CONTRACT_629AY
)

if not isinstance(
    historical_contract_630w10ar3,
    dict,
):
    raise RuntimeError(
        "FD003 TEST-admission authority is not a dict."
    )


required_historical_keys_630w10ar3 = (
    "window_length",
    "window_stride",
    "test_prediction_population",
    "short_engine_rule",
    "test_endpoint_order",
)

missing_historical_keys_630w10ar3 = [
    key
    for key in required_historical_keys_630w10ar3
    if key not in historical_contract_630w10ar3
]

if missing_historical_keys_630w10ar3:
    raise RuntimeError(
        "Historical TEST-admission contract is missing required key(s): "
        + ", ".join(
            missing_historical_keys_630w10ar3
        )
    )


historical_window_length_630w10ar3 = int(
    historical_contract_630w10ar3[
        "window_length"
    ]
)

historical_window_stride_630w10ar3 = int(
    historical_contract_630w10ar3[
        "window_stride"
    ]
)

historical_population_rule_630w10ar3 = str(
    historical_contract_630w10ar3[
        "test_prediction_population"
    ]
)

historical_short_engine_rule_630w10ar3 = str(
    historical_contract_630w10ar3[
        "short_engine_rule"
    ]
)

historical_endpoint_order_630w10ar3 = str(
    historical_contract_630w10ar3[
        "test_endpoint_order"
    ]
)


if historical_window_length_630w10ar3 != 30:
    raise RuntimeError(
        "Historical TEST window length is not 30."
    )

if historical_window_stride_630w10ar3 != 1:
    raise RuntimeError(
        "Historical TEST window stride is not 1."
    )


normalized_short_rule_630w10ar3 = (
    historical_short_engine_rule_630w10ar3
    .strip()
    .lower()
)

if not (
    "fewer than 30"
    in normalized_short_rule_630w10ar3
    and
    "stop before inference"
    in normalized_short_rule_630w10ar3
):
    raise RuntimeError(
        "Historical short-engine rule does not prove "
        "'stop before inference'."
    )


normalized_population_rule_630w10ar3 = (
    historical_population_rule_630w10ar3
    .strip()
    .lower()
)

if not (
    "one terminal 30-cycle window per test engine"
    in normalized_population_rule_630w10ar3
):
    raise RuntimeError(
        "Historical TEST population rule is not the expected "
        "one-terminal-30-cycle-window-per-engine protocol."
    )


# =============================================================================
# 3) VERIFY FD004 STRUCTURAL FACT
# =============================================================================

short_engine_count_630w10ar3 = int(
    FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
        "shorter_than_30_engine_count"
    ]
)

short_engine_lengths_630w10ar3 = tuple(
    (
        int(
            engine_id
        ),
        int(
            length
        ),
    )
    for engine_id, length
    in FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
        "shorter_than_30_engine_lengths"
    ]
)


if short_engine_count_630w10ar3 != len(
    short_engine_lengths_630w10ar3
):
    raise RuntimeError(
        "Short-engine count/list mismatch."
    )

if short_engine_count_630w10ar3 <= 0:
    raise RuntimeError(
        "R3 was entered even though no FD004 short TEST engine exists."
    )

if not all(
    length
    <
    historical_window_length_630w10ar3
    for _, length
    in short_engine_lengths_630w10ar3
):
    raise RuntimeError(
        "Short-engine evidence contains a trajectory that is not <30."
    )


test_engine_count_630w10ar3 = int(
    FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1[
        "test_engine_count"
    ]
)

structurally_eligible_engine_count_630w10ar3 = (
    test_engine_count_630w10ar3
    -
    short_engine_count_630w10ar3
)


# =============================================================================
# 4) APPLY HISTORICAL RULE — STOP BEFORE INFERENCE
#
# IMPORTANT:
# This is not a newly chosen FD004 rule.
# It is the already-existing historical admission rule.
# =============================================================================

FD004_PRIMARY_TEST_STRUCTURALLY_ADMISSIBLE_630W10AR3 = False

FD004_PRIMARY_TEST_INFERENCE_PERMITTED_630W10AR3 = False

FD004_PRIMARY_TEST_SCORING_PERMITTED_630W10AR3 = False

FD004_PRIMARY_TEST_STOPPED_BEFORE_INFERENCE_630W10AR3 = True


# =============================================================================
# 5) FREEZE STRUCTURAL-ADMISSIBILITY CONTRACT
# =============================================================================

FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-10A-R3",

    "status":
        "PRIMARY_HELDOUT_TEST_STOPPED_BEFORE_INFERENCE_SHORT_TRAJECTORY_GUARD",

    "test_contents_already_opened":
        bool(
            FD004_TEST_CONTENTS_READ_630W10A
        ),

    "official_rul_contents_already_opened":
        bool(
            FD004_RUL_CONTENTS_READ_630W10A
        ),

    "historical_method_source":
        "FD003_TEST_ADMISSION_CONTRACT_629AY",

    "historical_window_length":
        historical_window_length_630w10ar3,

    "historical_window_stride":
        historical_window_stride_630w10ar3,

    "historical_test_prediction_population":
        historical_population_rule_630w10ar3,

    "historical_short_engine_rule":
        historical_short_engine_rule_630w10ar3,

    "historical_endpoint_order":
        historical_endpoint_order_630w10ar3,

    "fd004_test_engine_count":
        test_engine_count_630w10ar3,

    "fd004_short_engine_count":
        short_engine_count_630w10ar3,

    "fd004_short_engine_lengths":
        short_engine_lengths_630w10ar3,

    "fd004_structurally_eligible_engine_count_if_subset_were_used":
        structurally_eligible_engine_count_630w10ar3,

    "primary_test_structurally_admissible":
        False,

    "primary_test_inference_permitted":
        False,

    "primary_test_scoring_permitted":
        False,

    "primary_test_stopped_before_inference":
        True,

    "reason":
        (
            "The inherited held-out TEST protocol requires one terminal "
            "30-cycle window for every TEST engine and explicitly requires "
            "stopping before inference if any TEST engine has fewer than "
            "30 observed cycles. FD004 contains such engines."
        ),

    "padding_applied":
        False,

    "row_repetition_applied":
        False,

    "short_engines_dropped":
        False,

    "variable_length_inference_used":
        False,

    "window_length_changed":
        False,

    "window_rule_changed":
        False,

    "test_subset_redefined":
        False,

    "model_inference_performed":
        False,

    "mc_dropout_executed":
        False,

    "test_risk_computed":
        False,

    "test_prediction_errors_computed":
        False,

    "test_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,

    "model_retrained":
        False,

    "preprocessing_refit":
        False,

    "risk_recalibrated":
        False,

    "primary_test_result":
        (
            "NOT_SCORED_UNDER_FROZEN_PROTOCOL_DUE_TO_SHORT_TEST_TRAJECTORIES"
        ),
}


FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3 = (
    _json_sha256_630w10ar3(
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3
    )
)


# =============================================================================
# 6) RESTART-SAFE CHECKPOINT
# =============================================================================

checkpoint_path_630w10ar3 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_test_structural_admissibility_630w10ar3_v1.json"
)

checkpoint_dir_630w10ar3 = os.path.dirname(
    checkpoint_path_630w10ar3
)

if checkpoint_dir_630w10ar3:
    os.makedirs(
        checkpoint_dir_630w10ar3,
        exist_ok=True,
    )


checkpoint_payload_630w10ar3 = {
    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3":
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3,

    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3":
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3,
}


tmp_path_630w10ar3 = (
    checkpoint_path_630w10ar3
    +
    ".tmp"
)

with open(
    tmp_path_630w10ar3,
    "w",
    encoding="utf-8",
) as handle_630w10ar3:
    json.dump(
        checkpoint_payload_630w10ar3,
        handle_630w10ar3,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )

os.replace(
    tmp_path_630w10ar3,
    checkpoint_path_630w10ar3,
)


with open(
    checkpoint_path_630w10ar3,
    "rb",
) as handle_630w10ar3:
    checkpoint_bytes_630w10ar3 = (
        handle_630w10ar3.read()
    )


FD004_TEST_STRUCTURAL_CHECKPOINT_FILE_SHA256_630W10AR3 = (
    hashlib.sha256(
        checkpoint_bytes_630w10ar3
    ).hexdigest()
)


with open(
    checkpoint_path_630w10ar3,
    "r",
    encoding="utf-8",
) as handle_630w10ar3:
    checkpoint_roundtrip_630w10ar3 = json.load(
        handle_630w10ar3
    )


checkpoint_roundtrip_exact_630w10ar3 = bool(
    checkpoint_roundtrip_630w10ar3
    ==
    checkpoint_payload_630w10ar3
)

if not checkpoint_roundtrip_exact_630w10ar3:
    raise RuntimeError(
        "Structural-admissibility checkpoint roundtrip failed."
    )


# =============================================================================
# 7) FINAL FLAGS
# =============================================================================

FD004_PRIMARY_TEST_EVALUATION_CLOSED_UNSCORED_630W10AR3 = True

FD004_TEST_METHOD_CHANGED_AFTER_ACCESS_630W10AR3 = False
FD004_TEST_TUNING_PERFORMED_630W10AR3 = False

all_checks_passed_630w10ar3 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-10A-R3 — FD004 PRIMARY HELD-OUT TEST "
    "STRUCTURAL-ADMISSIBILITY CLOSURE"
)

print("=" * 176)

print(
    "Historical authority              :",
    "FD003_TEST_ADMISSION_CONTRACT_629AY",
)

print(
    "Historical window length          :",
    historical_window_length_630w10ar3,
)

print(
    "Historical window stride          :",
    historical_window_stride_630w10ar3,
)

print(
    "Historical TEST population rule   :",
    historical_population_rule_630w10ar3,
)

print(
    "Historical short-engine rule      :",
    historical_short_engine_rule_630w10ar3,
)


print("\n--- FD004 STRUCTURAL FACT ---")

print(
    "TEST engine count                 :",
    test_engine_count_630w10ar3,
)

print(
    "Engines shorter than 30 cycles    :",
    short_engine_count_630w10ar3,
)

print(
    "Exact short engine lengths        :",
    short_engine_lengths_630w10ar3,
)

print(
    "Engines with >=30 cycles          :",
    structurally_eligible_engine_count_630w10ar3,
)


print("\n--- PRIMARY TEST DECISION ---")

print(
    "Structurally admissible           :",
    FD004_PRIMARY_TEST_STRUCTURALLY_ADMISSIBLE_630W10AR3,
)

print(
    "Inference permitted               :",
    FD004_PRIMARY_TEST_INFERENCE_PERMITTED_630W10AR3,
)

print(
    "Scoring permitted                 :",
    FD004_PRIMARY_TEST_SCORING_PERMITTED_630W10AR3,
)

print(
    "Stopped before inference          :",
    FD004_PRIMARY_TEST_STOPPED_BEFORE_INFERENCE_630W10AR3,
)


print("\n--- NO POST-TEST METHOD CHANGE ---")

print("Padding applied                   :", False)
print("Row repetition applied            :", False)
print("Short engines dropped             :", False)
print("Variable-length inference used    :", False)
print("Window length changed             :", False)
print("TEST population redefined         :", False)

print("Model inference performed         :", False)
print("MC dropout executed               :", False)
print("TEST risk computed                :", False)
print("TEST errors computed              :", False)
print("TEST metrics computed             :", False)

print("Threshold modified                :", False)
print("Policy modified                   :", False)
print("TEST tuning performed             :", False)


print("\n--- EVIDENCE LOCK ---")

print(
    "Structural contract SHA-256       :",
    FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3,
)

print(
    "Checkpoint path                   :",
    checkpoint_path_630w10ar3,
)

print(
    "Checkpoint file SHA-256           :",
    FD004_TEST_STRUCTURAL_CHECKPOINT_FILE_SHA256_630W10AR3,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_630w10ar3,
)


print(
    "\nPASS — the prospectively frozen primary FD004 TEST protocol "
    "has been honored exactly."
)

print(
    "Because FD004 contains TEST trajectories shorter than the frozen "
    "30-cycle window, the historical guard requires stopping before inference."
)

print(
    "No padding, engine exclusion, variable-length inference, or other "
    "post-TEST methodological repair was introduced."
)

print(
    "Primary FD004 held-out TEST status:"
)

print(
    "NOT SCORED under the frozen protocol because the complete "
    "248-engine endpoint population is structurally incompatible with "
    "the frozen 30-cycle input contract."
)

In [ ]:
# STEP 6.30W-10A-R3-R1 — STRUCTURAL-CLOSURE CHECKPOINT PERSISTENCE REPAIR
#
# Persistence repair ONLY.
#
# The R3 scientific contract was already created before the previous
# checkpoint roundtrip comparison failed.
#
# Cause of failure:
# JSON converts Python tuples to JSON arrays, which reload as Python lists.
# Direct Python equality therefore fails even when serialized content
# is scientifically identical.
#
# This repair:
# - verifies the already-created R3 structural contract + SHA
# - DOES NOT recompute the structural-admissibility decision
# - canonicalizes the checkpoint payload to JSON-native objects
# - writes a SAFE v2 checkpoint
# - verifies semantic JSON roundtrip exactly
# - completes the final R3 lock flags
#
# NO TEST inference.
# NO MC dropout.
# NO risk/errors/metrics.
# NO threshold/policy/model/preprocessing change.

import os
import json
import hashlib


# =============================================================================
# 0) REQUIRED RESIDENT R3 OBJECTS
# =============================================================================

required_630w10ar3r1 = (
    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3",
    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3",
)

missing_630w10ar3r1 = [
    name
    for name in required_630w10ar3r1
    if name not in globals()
]

if missing_630w10ar3r1:
    raise RuntimeError(
        "Resident R3 scientific object(s) missing: "
        + ", ".join(missing_630w10ar3r1)
    )


# =============================================================================
# 1) VERIFY THE ALREADY-CREATED R3 SCIENTIFIC CONTRACT
# =============================================================================

def _json_sha256_630w10ar3r1(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


contract_sha_recomputed_630w10ar3r1 = (
    _json_sha256_630w10ar3r1(
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3
    )
)

contract_sha_exact_630w10ar3r1 = bool(
    contract_sha_recomputed_630w10ar3r1
    ==
    str(
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3
    )
)

if not contract_sha_exact_630w10ar3r1:
    raise RuntimeError(
        "Resident R3 structural-admissibility contract changed."
    )


contract_630w10ar3r1 = (
    FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3
)


# Verify key scientific decision fields only; do not recompute them.
required_decision_fields_630w10ar3r1 = {
    "primary_test_structurally_admissible":
        False,

    "primary_test_inference_permitted":
        False,

    "primary_test_scoring_permitted":
        False,

    "primary_test_stopped_before_inference":
        True,

    "padding_applied":
        False,

    "row_repetition_applied":
        False,

    "short_engines_dropped":
        False,

    "variable_length_inference_used":
        False,

    "window_length_changed":
        False,

    "test_subset_redefined":
        False,

    "model_inference_performed":
        False,

    "mc_dropout_executed":
        False,

    "test_risk_computed":
        False,

    "test_prediction_errors_computed":
        False,

    "test_outcome_metrics_computed":
        False,

    "threshold_modified":
        False,

    "threshold_recomputed":
        False,

    "policy_modified":
        False,
}


for (
    key_630w10ar3r1,
    expected_630w10ar3r1,
) in required_decision_fields_630w10ar3r1.items():

    if key_630w10ar3r1 not in contract_630w10ar3r1:
        raise RuntimeError(
            "Resident R3 contract missing field: "
            + key_630w10ar3r1
        )

    if (
        contract_630w10ar3r1[
            key_630w10ar3r1
        ]
        !=
        expected_630w10ar3r1
    ):
        raise RuntimeError(
            "Resident R3 decision field changed: "
            + key_630w10ar3r1
        )


# =============================================================================
# 2) SHOW WHY THE ORIGINAL PYTHON ROUNDTRIP FAILED
# =============================================================================

short_lengths_resident_630w10ar3r1 = (
    contract_630w10ar3r1[
        "fd004_short_engine_lengths"
    ]
)

resident_short_lengths_type_630w10ar3r1 = type(
    short_lengths_resident_630w10ar3r1
).__name__

resident_nested_type_630w10ar3r1 = (
    type(
        short_lengths_resident_630w10ar3r1[0]
    ).__name__
    if len(
        short_lengths_resident_630w10ar3r1
    )
    >
    0
    else
    None
)


# =============================================================================
# 3) JSON-CANONICAL SAFE PAYLOAD
#
# json.dumps -> json.loads intentionally converts tuples to JSON-native lists.
# We compare canonical JSON objects rather than Python tuple/list identities.
# =============================================================================

canonical_contract_630w10ar3r1 = json.loads(
    json.dumps(
        contract_630w10ar3r1,
        sort_keys=True,
        allow_nan=False,
        default=str,
    )
)


safe_checkpoint_payload_630w10ar3r1 = {
    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3":
        canonical_contract_630w10ar3r1,

    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3":
        str(
            FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3
        ),

    "persistence_note":
        (
            "SAFE v2 persistence repair only; JSON-native canonicalization "
            "converts Python tuples to JSON arrays/lists without changing "
            "the scientific structural-admissibility decision."
        ),
}


# =============================================================================
# 4) WRITE SAFE V2 CHECKPOINT
# =============================================================================

safe_checkpoint_path_630w10ar3r1 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_test_structural_admissibility_630w10ar3_v2_safe.json"
)

safe_checkpoint_dir_630w10ar3r1 = os.path.dirname(
    safe_checkpoint_path_630w10ar3r1
)

if safe_checkpoint_dir_630w10ar3r1:
    os.makedirs(
        safe_checkpoint_dir_630w10ar3r1,
        exist_ok=True,
    )


safe_tmp_path_630w10ar3r1 = (
    safe_checkpoint_path_630w10ar3r1
    +
    ".tmp"
)

with open(
    safe_tmp_path_630w10ar3r1,
    "w",
    encoding="utf-8",
) as handle_630w10ar3r1:

    json.dump(
        safe_checkpoint_payload_630w10ar3r1,
        handle_630w10ar3r1,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


os.replace(
    safe_tmp_path_630w10ar3r1,
    safe_checkpoint_path_630w10ar3r1,
)


# =============================================================================
# 5) VERIFY FILE SHA + SEMANTIC ROUNDTRIP
# =============================================================================

with open(
    safe_checkpoint_path_630w10ar3r1,
    "rb",
) as handle_630w10ar3r1:

    safe_checkpoint_bytes_630w10ar3r1 = (
        handle_630w10ar3r1.read()
    )


FD004_TEST_STRUCTURAL_CHECKPOINT_FILE_SHA256_630W10AR3R1 = (
    hashlib.sha256(
        safe_checkpoint_bytes_630w10ar3r1
    ).hexdigest()
)


with open(
    safe_checkpoint_path_630w10ar3r1,
    "r",
    encoding="utf-8",
) as handle_630w10ar3r1:

    safe_roundtrip_630w10ar3r1 = json.load(
        handle_630w10ar3r1
    )


safe_roundtrip_exact_630w10ar3r1 = bool(
    safe_roundtrip_630w10ar3r1
    ==
    safe_checkpoint_payload_630w10ar3r1
)

if not safe_roundtrip_exact_630w10ar3r1:
    raise RuntimeError(
        "SAFE v2 structural checkpoint JSON roundtrip failed."
    )


# Also prove the reloaded scientific contract is semantically identical
# to the JSON-canonical resident scientific contract.

reloaded_contract_630w10ar3r1 = (
    safe_roundtrip_630w10ar3r1[
        "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3"
    ]
)

scientific_contract_semantic_exact_630w10ar3r1 = bool(
    reloaded_contract_630w10ar3r1
    ==
    canonical_contract_630w10ar3r1
)

if not scientific_contract_semantic_exact_630w10ar3r1:
    raise RuntimeError(
        "Reloaded SAFE v2 scientific contract is not semantically exact."
    )


# Verify original scientific SHA still unchanged after persistence repair.
contract_sha_after_repair_630w10ar3r1 = (
    _json_sha256_630w10ar3r1(
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3
    )
)

if (
    contract_sha_after_repair_630w10ar3r1
    !=
    str(
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3
    )
):
    raise RuntimeError(
        "Scientific R3 contract changed during persistence repair."
    )


# =============================================================================
# 6) COMPLETE AUTHORITATIVE R3 LOCK FLAGS
# =============================================================================

FD004_PRIMARY_TEST_EVALUATION_CLOSED_UNSCORED_630W10AR3 = True

FD004_TEST_METHOD_CHANGED_AFTER_ACCESS_630W10AR3 = False
FD004_TEST_TUNING_PERFORMED_630W10AR3 = False

FD004_TEST_STRUCTURAL_CHECKPOINT_PATH_630W10AR3R1 = (
    safe_checkpoint_path_630w10ar3r1
)

FD004_TEST_STRUCTURAL_CHECKPOINT_ROUNDTRIP_EXACT_630W10AR3R1 = (
    True
)

FD004_TEST_STRUCTURAL_SCIENTIFIC_CONTRACT_UNCHANGED_630W10AR3R1 = (
    True
)

all_checks_passed_630w10ar3 = True
all_checks_passed_630w10ar3r1 = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-10A-R3-R1 — FD004 STRUCTURAL-CLOSURE "
    "CHECKPOINT PERSISTENCE REPAIR"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("TEST inference performed          :", False)
print("MC dropout executed               :", False)
print("TEST risk computed                :", False)
print("TEST errors computed              :", False)
print("TEST metrics computed             :", False)
print("Threshold modified                :", False)
print("Policy modified                   :", False)


print("\n--- ORIGINAL R3 CONTRACT ---")

print(
    "Scientific contract SHA exact     :",
    contract_sha_exact_630w10ar3r1,
)

print(
    "Resident short-length container   :",
    resident_short_lengths_type_630w10ar3r1,
)

print(
    "Resident nested container         :",
    resident_nested_type_630w10ar3r1,
)

print(
    "Failure explanation               :",
    (
        "JSON arrays reload as Python lists, so tuple/list "
        "Python equality failed in the original checkpoint check."
    ),
)


print("\n--- STRUCTURAL DECISION PRESERVED ---")

print(
    "Primary TEST admissible           :",
    contract_630w10ar3r1[
        "primary_test_structurally_admissible"
    ],
)

print(
    "Inference permitted               :",
    contract_630w10ar3r1[
        "primary_test_inference_permitted"
    ],
)

print(
    "Scoring permitted                 :",
    contract_630w10ar3r1[
        "primary_test_scoring_permitted"
    ],
)

print(
    "Stopped before inference          :",
    contract_630w10ar3r1[
        "primary_test_stopped_before_inference"
    ],
)

print(
    "Short TEST engine count           :",
    contract_630w10ar3r1[
        "fd004_short_engine_count"
    ],
)

print(
    "Primary TEST result               :",
    contract_630w10ar3r1[
        "primary_test_result"
    ],
)


print("\n--- SAFE V2 PERSISTENCE ---")

print(
    "Checkpoint path                   :",
    safe_checkpoint_path_630w10ar3r1,
)

print(
    "Checkpoint file SHA-256           :",
    FD004_TEST_STRUCTURAL_CHECKPOINT_FILE_SHA256_630W10AR3R1,
)

print(
    "JSON roundtrip exact              :",
    safe_roundtrip_exact_630w10ar3r1,
)

print(
    "Scientific contract semantic exact:",
    scientific_contract_semantic_exact_630w10ar3r1,
)

print(
    "Scientific SHA unchanged          :",
    (
        contract_sha_after_repair_630w10ar3r1
        ==
        str(
            FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3
        )
    ),
)


print(
    "\nPASS — the previous failure was persistence-only."
)

print(
    "The R3 scientific structural-admissibility decision is unchanged "
    "and is now restart-safe in the SAFE v2 checkpoint."
)

print(
    "No TEST inference or outcome calculation occurred."
)

In [ ]:
# STEP 6.30W-11 — FD004 FINAL DATASET CLOSURE / REPORTING-ONLY EVIDENCE LOCK
#
# Final evidence state:
# - CLEAN DEV selective result: completed and frozen
# - 381 controlled-corruption scenarios: completed and frozen
# - pre-TEST evidence: closed before TEST access
# - one-way TEST admission: completed
# - TEST/RUL: opened for fixed evaluation
# - primary held-out TEST inference: NOT PERFORMED
# - reason: inherited protocol requires one terminal 30-cycle window per
#   TEST engine and explicitly says to stop before inference if any TEST
#   engine has fewer than 30 observed cycles
# - FD004 has 11 such engines
#
# This cell:
# - verifies decisive evidence locks
# - freezes the final FD004 scientific conclusion
# - freezes the final dataset closure contract
# - writes a restart-safe final checkpoint
#
# NO scientific recomputation.
# NO TEST inference.
# NO TEST risk/error/MAE.
# NO threshold/policy/model/preprocessing change.

import os
import json
import hashlib


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_630w11 = (
    "all_checks_passed_630w10ar3r1",

    "FD004_PRETEST_EVIDENCE_CLOSURE_630W8",
    "FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8",

    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9",
    "FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9",

    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1",
    "FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1",

    "FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_630W10AR2",
    "FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2",

    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3",
    "FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3",

    "FD004_TEST_STRUCTURAL_CHECKPOINT_FILE_SHA256_630W10AR3R1",
    "FD004_TEST_STRUCTURAL_CHECKPOINT_PATH_630W10AR3R1",

    "FD004_PRIMARY_TEST_EVALUATION_CLOSED_UNSCORED_630W10AR3",
    "FD004_TEST_METHOD_CHANGED_AFTER_ACCESS_630W10AR3",
    "FD004_TEST_TUNING_PERFORMED_630W10AR3",
)

missing_630w11 = [
    name
    for name in required_630w11
    if name not in globals()
]

if missing_630w11:
    raise RuntimeError(
        "Missing STEP 6.30W-11 authority object(s): "
        + ", ".join(missing_630w11)
    )

if not bool(all_checks_passed_630w10ar3r1):
    raise RuntimeError(
        "STEP 6.30W-10A-R3-R1 must remain passed."
    )


# =============================================================================
# 1) HASH VERIFICATION
# =============================================================================

def _json_sha256_630w11(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


hash_authorities_630w11 = (
    (
        "pretest evidence closure",
        FD004_PRETEST_EVIDENCE_CLOSURE_630W8,
        FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8,
    ),
    (
        "one-way TEST admission",
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_630W9,
        FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9,
    ),
    (
        "short-trajectory recovery audit",
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_630W10AR1,
        FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1,
    ),
    (
        "targeted historical-rule recovery audit",
        FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_630W10AR2,
        FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2,
    ),
    (
        "TEST structural-admissibility contract",
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3,
        FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3,
    ),
)

hash_checks_630w11 = {}

for (
    label_630w11,
    obj_630w11,
    expected_sha_630w11,
) in hash_authorities_630w11:

    actual_sha_630w11 = _json_sha256_630w11(
        obj_630w11
    )

    passed_630w11 = bool(
        actual_sha_630w11
        ==
        str(expected_sha_630w11)
    )

    hash_checks_630w11[
        label_630w11
    ] = passed_630w11

    if not passed_630w11:
        raise RuntimeError(
            "Frozen evidence hash changed: "
            + label_630w11
        )


# =============================================================================
# 2) VERIFY FINAL TEST STATE
# =============================================================================

structural_630w11 = (
    FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_630W10AR3
)

if bool(
    structural_630w11[
        "primary_test_structurally_admissible"
    ]
):
    raise RuntimeError(
        "FD004 primary TEST unexpectedly became structurally admissible."
    )

if bool(
    structural_630w11[
        "primary_test_inference_permitted"
    ]
):
    raise RuntimeError(
        "FD004 primary TEST inference unexpectedly became permitted."
    )

if bool(
    structural_630w11[
        "primary_test_scoring_permitted"
    ]
):
    raise RuntimeError(
        "FD004 primary TEST scoring unexpectedly became permitted."
    )

if not bool(
    structural_630w11[
        "primary_test_stopped_before_inference"
    ]
):
    raise RuntimeError(
        "FD004 primary TEST is not marked stopped before inference."
    )


for key_630w11 in (
    "padding_applied",
    "row_repetition_applied",
    "short_engines_dropped",
    "variable_length_inference_used",
    "window_length_changed",
    "window_rule_changed",
    "test_subset_redefined",
    "model_inference_performed",
    "mc_dropout_executed",
    "test_risk_computed",
    "test_prediction_errors_computed",
    "test_outcome_metrics_computed",
    "threshold_modified",
    "threshold_recomputed",
    "policy_modified",
    "model_retrained",
    "preprocessing_refit",
    "risk_recalibrated",
):
    if bool(
        structural_630w11[
            key_630w11
        ]
    ):
        raise RuntimeError(
            "Final structural guard changed: "
            + key_630w11
        )


if not bool(
    FD004_PRIMARY_TEST_EVALUATION_CLOSED_UNSCORED_630W10AR3
):
    raise RuntimeError(
        "Primary FD004 TEST is not marked closed/unscored."
    )

if bool(
    FD004_TEST_METHOD_CHANGED_AFTER_ACCESS_630W10AR3
):
    raise RuntimeError(
        "A post-access TEST method change was recorded."
    )

if bool(
    FD004_TEST_TUNING_PERFORMED_630W10AR3
):
    raise RuntimeError(
        "Post-access TEST tuning was recorded."
    )


# =============================================================================
# 3) BIND FROZEN DEV / CORRUPTION EVIDENCE
# =============================================================================

pretest_630w11 = (
    FD004_PRETEST_EVIDENCE_CLOSURE_630W8
)

clean_630w11 = (
    pretest_630w11[
        "clean_dev"
    ]
)

corruption_630w11 = (
    pretest_630w11[
        "controlled_corruption"
    ]
)

policy_630w11 = (
    pretest_630w11[
        "policy"
    ]
)


short_engine_count_630w11 = int(
    structural_630w11[
        "fd004_short_engine_count"
    ]
)

test_engine_count_630w11 = int(
    structural_630w11[
        "fd004_test_engine_count"
    ]
)

short_engine_lengths_630w11 = tuple(
    (
        int(engine_id_630w11),
        int(length_630w11),
    )
    for engine_id_630w11, length_630w11
    in structural_630w11[
        "fd004_short_engine_lengths"
    ]
)


if short_engine_count_630w11 != 11:
    raise RuntimeError(
        "Final FD004 short-engine count changed."
    )

if test_engine_count_630w11 != 248:
    raise RuntimeError(
        "Final FD004 TEST engine count changed."
    )


# =============================================================================
# 4) FINAL SCIENTIFIC CONCLUSION
# =============================================================================

FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11 = {
    "dataset":
        "FD004",

    "clean_dev_interpretation":
        (
            "Under the prospectively frozen label-free operating policy, "
            "CLEAN DEV selective prediction improved MAE at approximately "
            "80% realized coverage: accepted predictions had lower MAE than "
            "the full DEV population, while abstained predictions had "
            "substantially higher MAE."
        ),

    "controlled_corruption_interpretation":
        (
            "Across the 381 executable controlled sensor-degradation "
            "scenarios, the frozen selective policy produced positive "
            "selective MAE gain in 365 scenarios. The 16 negative-gain "
            "scenarios all occurred in the stuck-sensor family, with the "
            "limitation concentrated most strongly at high severity."
        ),

    "heldout_test_interpretation":
        (
            "No primary held-out FD004 TEST performance metric is reported. "
            "The inherited prospectively frozen TEST protocol requires one "
            "terminal 30-cycle window for every TEST engine and explicitly "
            "requires stopping before inference if any TEST engine has fewer "
            "than 30 observed cycles. FD004 contains 11 such engines among "
            "248 TEST engines, so primary TEST inference/scoring was stopped "
            "before prediction rather than introducing post-access padding, "
            "engine exclusion, variable-length inference, or another "
            "methodological repair."
        ),

    "generalization_claim":
        (
            "FD004 supports the DEV-stage and controlled-degradation evidence "
            "for reliability-aware selective RUL prediction, including a "
            "documented stuck-sensor failure mode. It does not provide an "
            "FD004 held-out selective-performance estimate under the frozen "
            "primary TEST protocol because that protocol is structurally "
            "incompatible with the complete TEST endpoint population."
        ),

    "methodological_integrity":
        (
            "TEST and official RUL were opened only after the pre-TEST "
            "evidence closure and one-way admission gate. No deterministic "
            "TEST inference, MC-dropout uncertainty, TEST risk, prediction "
            "error, MAE, threshold retuning, policy change, model retraining, "
            "preprocessing refit, or risk recalibration was performed after "
            "the short-trajectory guard was triggered."
        ),

    "clean_dev": {
        "realized_coverage":
            float(
                clean_630w11[
                    "realized_coverage"
                ]
            ),

        "full_mae":
            float(
                clean_630w11[
                    "full_mae"
                ]
            ),

        "accepted_mae":
            float(
                clean_630w11[
                    "accepted_mae"
                ]
            ),

        "abstained_mae":
            float(
                clean_630w11[
                    "abstained_mae"
                ]
            ),

        "selective_mae_gain":
            float(
                clean_630w11[
                    "selective_mae_gain"
                ]
            ),

        "relative_mae_reduction_percent":
            float(
                clean_630w11[
                    "relative_mae_reduction_percent"
                ]
            ),
    },

    "controlled_corruption": {
        "scenario_count":
            int(
                corruption_630w11[
                    "scenario_count"
                ]
            ),

        "positive_gain_count":
            int(
                corruption_630w11[
                    "positive_gain_count"
                ]
            ),

        "zero_gain_count":
            int(
                corruption_630w11[
                    "zero_gain_count"
                ]
            ),

        "negative_gain_count":
            int(
                corruption_630w11[
                    "negative_gain_count"
                ]
            ),

        "positive_gain_rate":
            float(
                corruption_630w11[
                    "positive_gain_rate"
                ]
            ),

        "negative_gain_family":
            str(
                corruption_630w11[
                    "negative_gain_family"
                ]
            ),

        "negative_gain_by_severity":
            dict(
                corruption_630w11[
                    "negative_gain_by_severity"
                ]
            ),
    },

    "heldout_test": {
        "test_engine_count":
            test_engine_count_630w11,

        "frozen_window_length":
            int(
                structural_630w11[
                    "historical_window_length"
                ]
            ),

        "short_engine_count":
            short_engine_count_630w11,

        "short_engine_lengths":
            short_engine_lengths_630w11,

        "structurally_admissible":
            False,

        "inference_performed":
            False,

        "scoring_performed":
            False,

        "primary_result":
            str(
                structural_630w11[
                    "primary_test_result"
                ]
            ),
    },
}


FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11 = (
    _json_sha256_630w11(
        FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11
    )
)


# =============================================================================
# 5) FINAL DATASET CLOSURE CONTRACT
# =============================================================================

FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11 = {
    "dataset":
        "FD004",

    "stage":
        "6.30W-11",

    "status":
        "COMPLETE_REPORTING_ONLY_EVIDENCE_LOCKED_TEST_STRUCTURALLY_UNSCORED",

    "pretest_evidence_closure_sha256":
        str(
            FD004_PRETEST_EVIDENCE_CLOSURE_SHA256_630W8
        ),

    "one_way_test_admission_sha256":
        str(
            FD004_ONE_WAY_TEST_ADMISSION_CONTRACT_SHA256_630W9
        ),

    "short_trajectory_recovery_sha256":
        str(
            FD004_SHORT_TEST_METHOD_RECOVERY_AUDIT_SHA256_630W10AR1
        ),

    "targeted_historical_rule_recovery_sha256":
        str(
            FD004_TARGETED_TEST_RULE_RECOVERY_AUDIT_SHA256_630W10AR2
        ),

    "test_structural_admissibility_sha256":
        str(
            FD004_TEST_STRUCTURAL_ADMISSIBILITY_CONTRACT_SHA256_630W10AR3
        ),

    "final_scientific_conclusion_sha256":
        str(
            FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11
        ),

    "frozen_policy": {
        "nominal_coverage":
            float(
                policy_630w11[
                    "nominal_coverage"
                ]
            ),

        "risk_threshold":
            float(
                policy_630w11[
                    "frozen_risk_threshold"
                ]
            ),

        "acceptance_rule":
            str(
                policy_630w11[
                    "acceptance_rule"
                ]
            ),
    },

    "clean_dev_completed":
        True,

    "controlled_corruption_completed":
        True,

    "controlled_corruption_scenario_count":
        381,

    "test_admitted_one_way":
        True,

    "test_contents_opened":
        True,

    "official_rul_contents_opened":
        True,

    "primary_test_structurally_admissible":
        False,

    "primary_test_inference_performed":
        False,

    "primary_test_scored":
        False,

    "primary_test_unscored_reason":
        (
            "11 of 248 TEST engines have fewer than the frozen 30 "
            "observed cycles required by the inherited terminal-window "
            "TEST protocol; historical rule requires stopping before inference"
        ),

    "short_test_engine_count":
        short_engine_count_630w11,

    "short_test_engine_lengths":
        short_engine_lengths_630w11,

    "post_test_padding":
        False,

    "post_test_engine_exclusion":
        False,

    "post_test_variable_length_inference":
        False,

    "post_test_window_rule_change":
        False,

    "threshold_change_after_test":
        False,

    "policy_change_after_test":
        False,

    "model_retraining_after_test":
        False,

    "preprocessing_refit_after_test":
        False,

    "risk_recalibration_after_test":
        False,

    "test_outcome_metrics_computed":
        False,

    "reporting_rule":
        (
            "Report FD004 CLEAN DEV and controlled-corruption findings as "
            "completed evidence; report primary held-out TEST as structurally "
            "unscored under the frozen protocol; do not substitute a post-hoc "
            "237-engine subset or padded/variable-length TEST analysis as the "
            "primary held-out result."
        ),

    "next_stage":
        "CROSS_DATASET_SYNTHESIS_FD001_FD004",
}


FD004_FINAL_DATASET_CLOSURE_SHA256_630W11 = (
    _json_sha256_630w11(
        FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11
    )
)


# =============================================================================
# 6) JSON-CANONICAL RESTART-SAFE FINAL CHECKPOINT
# =============================================================================

canonical_final_conclusion_630w11 = json.loads(
    json.dumps(
        FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11,
        sort_keys=True,
        allow_nan=False,
        default=str,
    )
)

canonical_final_closure_630w11 = json.loads(
    json.dumps(
        FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11,
        sort_keys=True,
        allow_nan=False,
        default=str,
    )
)


final_checkpoint_payload_630w11 = {
    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11":
        canonical_final_conclusion_630w11,

    "FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11":
        str(
            FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11
        ),

    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11":
        canonical_final_closure_630w11,

    "FD004_FINAL_DATASET_CLOSURE_SHA256_630W11":
        str(
            FD004_FINAL_DATASET_CLOSURE_SHA256_630W11
        ),
}


final_checkpoint_path_630w11 = (
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd004_final_dataset_closure_630w11_v1_safe.json"
)

final_checkpoint_dir_630w11 = os.path.dirname(
    final_checkpoint_path_630w11
)

if final_checkpoint_dir_630w11:
    os.makedirs(
        final_checkpoint_dir_630w11,
        exist_ok=True,
    )


final_tmp_path_630w11 = (
    final_checkpoint_path_630w11
    +
    ".tmp"
)

with open(
    final_tmp_path_630w11,
    "w",
    encoding="utf-8",
) as handle_630w11:

    json.dump(
        final_checkpoint_payload_630w11,
        handle_630w11,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


os.replace(
    final_tmp_path_630w11,
    final_checkpoint_path_630w11,
)


with open(
    final_checkpoint_path_630w11,
    "rb",
) as handle_630w11:

    final_checkpoint_bytes_630w11 = (
        handle_630w11.read()
    )


FD004_FINAL_DATASET_CHECKPOINT_FILE_SHA256_630W11 = (
    hashlib.sha256(
        final_checkpoint_bytes_630w11
    ).hexdigest()
)


with open(
    final_checkpoint_path_630w11,
    "r",
    encoding="utf-8",
) as handle_630w11:

    final_roundtrip_630w11 = json.load(
        handle_630w11
    )


final_roundtrip_exact_630w11 = bool(
    final_roundtrip_630w11
    ==
    final_checkpoint_payload_630w11
)

if not final_roundtrip_exact_630w11:
    raise RuntimeError(
        "Final FD004 closure checkpoint roundtrip failed."
    )


# =============================================================================
# 7) FINAL LOCK FLAGS
# =============================================================================

FD004_FINAL_DATASET_CLOSURE_LOCKED_630W11 = True
FD004_CLOSED_630W11 = True
FD004_REPORTING_ONLY_AFTER_CLOSURE_630W11 = True
FD004_READY_FOR_CROSS_DATASET_SYNTHESIS_630W11 = True

all_checks_passed_630w11 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.30W-11 — FD004 FINAL DATASET CLOSURE / "
    "REPORTING-ONLY EVIDENCE LOCK"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("TEST inference performed          :", False)
print("MC dropout executed               :", False)
print("TEST risk/errors/metrics computed :", False)
print("Threshold modified                :", False)
print("Policy modified                   :", False)
print("Model/preprocessing changed       :", False)


print("\n--- CLEAN DEV FINAL EVIDENCE ---")

print(
    "Coverage                          :",
    clean_630w11[
        "realized_coverage"
    ],
)

print(
    "Full / accepted / abstained MAE   :",
    (
        clean_630w11[
            "full_mae"
        ],
        clean_630w11[
            "accepted_mae"
        ],
        clean_630w11[
            "abstained_mae"
        ],
    ),
)

print(
    "Selective gain                    :",
    clean_630w11[
        "selective_mae_gain"
    ],
)

print(
    "Relative MAE reduction (%)        :",
    clean_630w11[
        "relative_mae_reduction_percent"
    ],
)


print("\n--- CONTROLLED CORRUPTION FINAL EVIDENCE ---")

print(
    "Scenarios                         :",
    corruption_630w11[
        "scenario_count"
    ],
)

print(
    "Positive / zero / negative gain   :",
    (
        corruption_630w11[
            "positive_gain_count"
        ],
        corruption_630w11[
            "zero_gain_count"
        ],
        corruption_630w11[
            "negative_gain_count"
        ],
    ),
)

print(
    "Positive-gain rate                :",
    corruption_630w11[
        "positive_gain_rate"
    ],
)

print(
    "Negative-gain family              :",
    corruption_630w11[
        "negative_gain_family"
    ],
)

print(
    "Negative gain by severity         :",
    corruption_630w11[
        "negative_gain_by_severity"
    ],
)


print("\n--- HELD-OUT TEST FINAL STATUS ---")

print(
    "TEST engines                      :",
    test_engine_count_630w11,
)

print(
    "Short engines (<30 cycles)        :",
    short_engine_count_630w11,
)

print(
    "Short engine lengths              :",
    short_engine_lengths_630w11,
)

print(
    "Primary TEST structurally valid   :",
    False,
)

print(
    "Primary TEST inference performed  :",
    False,
)

print(
    "Primary TEST scored               :",
    False,
)

print(
    "Primary TEST status               :",
    structural_630w11[
        "primary_test_result"
    ],
)


print("\n--- FINAL SCIENTIFIC INTERPRETATION ---")

print(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "clean_dev_interpretation"
    ]
)

print()

print(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "controlled_corruption_interpretation"
    ]
)

print()

print(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "heldout_test_interpretation"
    ]
)


print("\n--- EVIDENCE LOCK ---")

print(
    "All source hashes exact           :",
    all(
        hash_checks_630w11.values()
    ),
)

print(
    "Final conclusion SHA-256          :",
    FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11,
)

print(
    "Final dataset closure SHA-256     :",
    FD004_FINAL_DATASET_CLOSURE_SHA256_630W11,
)

print(
    "Final checkpoint path             :",
    final_checkpoint_path_630w11,
)

print(
    "Final checkpoint file SHA-256     :",
    FD004_FINAL_DATASET_CHECKPOINT_FILE_SHA256_630W11,
)

print(
    "Final checkpoint roundtrip exact  :",
    final_roundtrip_exact_630w11,
)

print(
    "FD004 closed                      :",
    FD004_CLOSED_630W11,
)

print(
    "Ready for cross-dataset synthesis :",
    FD004_READY_FOR_CROSS_DATASET_SYNTHESIS_630W11,
)


print(
    "\nPASS — FD004 is permanently closed for scientific execution."
)

print(
    "All subsequent FD004 use is reporting/synthesis only."
)

print(
    "Next: synthesize FD001–FD004 evidence and prepare the final "
    "cross-dataset results, discussion, limitations, tables, and figures."
)

In [ ]:
# STEP 6.31A — FD001–FD004 CROSS-DATASET FINAL AUTHORITY LOCATOR
#
# Purpose:
# Locate the FINAL resident reporting/closure authorities for FD001–FD004
# before constructing any cross-dataset synthesis table.
#
# This is REPORTING-ONLY / STATIC INSPECTION.
#
# NO model inference.
# NO MC dropout.
# NO new corruption execution.
# NO threshold/policy selection.
# NO scientific recomputation.
#
# We deliberately do NOT assume historical FD001/FD002 final object names.

import json
import hashlib
import numbers
import re


# =============================================================================
# 0) PRECONDITION — FD004 MUST BE CLOSED
# =============================================================================

required_631a = (
    "all_checks_passed_630w11",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
    "FD004_FINAL_DATASET_CLOSURE_SHA256_630W11",
    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
    "FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11",
)

missing_631a = [
    name
    for name in required_631a
    if name not in globals()
]

if missing_631a:
    raise RuntimeError(
        "Missing STEP 6.31A authority object(s): "
        + ", ".join(missing_631a)
    )

if not bool(all_checks_passed_630w11):
    raise RuntimeError(
        "FD004 final closure must remain passed."
    )


# =============================================================================
# 1) VERIFY FD004 FINAL LOCK
# =============================================================================

def _json_sha256_631a(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631a(
        FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11
    )
    !=
    str(
        FD004_FINAL_DATASET_CLOSURE_SHA256_630W11
    )
):
    raise RuntimeError(
        "FD004 final closure changed before synthesis."
    )

if (
    _json_sha256_631a(
        FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11
    )
    !=
    str(
        FD004_FINAL_SCIENTIFIC_CONCLUSION_SHA256_630W11
    )
):
    raise RuntimeError(
        "FD004 final scientific conclusion changed before synthesis."
    )


# =============================================================================
# 2) CANDIDATE OBJECT SCORING
# =============================================================================

dataset_tokens_631a = (
    "FD001",
    "FD002",
    "FD003",
    "FD004",
)

name_tokens_631a = (
    "FINAL",
    "CLOSURE",
    "CONCLUSION",
    "TEST",
    "HELDOUT",
    "REPORT",
    "OUTCOME",
    "POLICY",
    "SELECTIVE",
    "SCORE",
    "EVIDENCE",
)


def _candidate_score_631a(
    name,
    value,
):
    upper = name.upper()

    score = 0

    if "FINAL" in upper:
        score += 40

    if "CLOSURE" in upper:
        score += 40

    if "CONCLUSION" in upper:
        score += 35

    if "HELDOUT" in upper:
        score += 30

    if "TEST" in upper:
        score += 25

    if "REPORT" in upper:
        score += 20

    if "OUTCOME" in upper:
        score += 20

    if "SELECTIVE" in upper:
        score += 15

    if "SCORE" in upper:
        score += 15

    if "POLICY" in upper:
        score += 10

    if isinstance(
        value,
        dict,
    ):
        lower_keys = {
            str(k).lower()
            for k in value.keys()
        }

        for key, weight in (
            ("status", 15),
            ("full_mae", 15),
            ("accepted_mae", 15),
            ("abstained_mae", 15),
            ("coverage", 10),
            ("realized_coverage", 10),
            ("selective_mae_gain", 15),
            ("relative_mae_reduction_percent", 15),
            ("heldout_test", 20),
            ("clean_dev", 15),
            ("controlled_corruption", 15),
            ("primary_result", 15),
        ):
            if key in lower_keys:
                score += weight

    return score


# =============================================================================
# 3) COMPACT RECURSIVE FIELD EXTRACTION
# =============================================================================

field_re_631a = re.compile(
    r"status|"
    r"coverage|"
    r"full_mae|"
    r"accepted_mae|"
    r"abstained_mae|"
    r"selective.*gain|"
    r"relative.*reduction|"
    r"threshold|"
    r"endpoint_count|"
    r"test_engine_count|"
    r"scenario_count|"
    r"positive_gain|"
    r"negative_gain|"
    r"primary_result|"
    r"structurally_admissible|"
    r"scored|"
    r"inference_performed",
    flags=re.IGNORECASE,
)


def _scalar_repr_631a(value):
    if isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        return repr(
            value
        )

    return None


def _extract_fields_631a(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 6:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():

            key_text = str(
                key
            )

            new_path = (
                path
                +
                (
                    key_text,
                )
            )

            scalar = _scalar_repr_631a(
                child
            )

            path_text = ".".join(
                new_path
            )

            if (
                scalar is not None
                and
                field_re_631a.search(
                    path_text
                )
            ):
                output.append(
                    (
                        path_text,
                        scalar,
                    )
                )

            _extract_fields_631a(
                child,
                new_path,
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) <= 30:
            for idx, child in enumerate(
                value
            ):
                _extract_fields_631a(
                    child,
                    (
                        path
                        +
                        (
                            f"[{idx}]",
                        )
                    ),
                    depth + 1,
                    output,
                )

    return output


# =============================================================================
# 4) LOCATE FINAL REPORTING AUTHORITIES BY DATASET
# =============================================================================

FD004_CROSS_DATASET_AUTHORITY_CANDIDATES_631A = {}

for dataset_631a in dataset_tokens_631a:

    candidates_631a = []

    for name_631a, value_631a in list(
        globals().items()
    ):

        upper_name_631a = (
            name_631a.upper()
        )

        if dataset_631a not in upper_name_631a:
            continue

        if not any(
            token in upper_name_631a
            for token in name_tokens_631a
        ):
            continue

        if not isinstance(
            value_631a,
            dict,
        ):
            continue

        score_631a = _candidate_score_631a(
            name_631a,
            value_631a,
        )

        if score_631a <= 0:
            continue

        extracted_631a = _extract_fields_631a(
            value_631a
        )

        candidates_631a.append(
            {
                "name":
                    name_631a,

                "score":
                    int(
                        score_631a
                    ),

                "top_level_keys":
                    tuple(
                        str(k)
                        for k in value_631a.keys()
                    ),

                "selected_fields":
                    tuple(
                        extracted_631a[
                            :50
                        ]
                    ),
            }
        )


    candidates_631a = sorted(
        candidates_631a,
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "name"
            ],
        ),
    )

    FD004_CROSS_DATASET_AUTHORITY_CANDIDATES_631A[
        dataset_631a
    ] = tuple(
        candidates_631a[
            :12
        ]
    )


# =============================================================================
# 5) CHECK THAT EACH DATASET HAS AT LEAST ONE REPORTING AUTHORITY
# =============================================================================

dataset_candidate_counts_631a = {
    dataset:
        len(
            FD004_CROSS_DATASET_AUTHORITY_CANDIDATES_631A[
                dataset
            ]
        )
    for dataset
    in dataset_tokens_631a
}


datasets_without_candidates_631a = tuple(
    dataset
    for dataset, count
    in dataset_candidate_counts_631a.items()
    if count == 0
)


# Do not fail if a historical dataset has no obvious candidate:
# this is precisely what the locator is designed to discover.


# =============================================================================
# 6) FREEZE LOCATOR AUDIT
# =============================================================================

FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_631A = {
    "stage":
        "6.31A",

    "status":
        "FD001_FD004_FINAL_AUTHORITY_LOCATOR_COMPLETE",

    "dataset_candidate_counts":
        dataset_candidate_counts_631a,

    "datasets_without_candidates":
        datasets_without_candidates_631a,

    "fd004_final_closure_verified":
        True,

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "new_corruption_execution":
        False,

    "threshold_selection":
        False,

    "policy_selection":
        False,

    "reporting_only":
        True,
}


FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_SHA256_631A = (
    _json_sha256_631a(
        FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_631A
    )
)

all_checks_passed_631a = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31A — FD001–FD004 CROSS-DATASET "
    "FINAL AUTHORITY LOCATOR"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("New corruption execution          :", False)
print("Threshold/policy selection        :", False)
print("Reporting-only inspection         :", True)


for dataset_631a in dataset_tokens_631a:

    candidates_631a = (
        FD004_CROSS_DATASET_AUTHORITY_CANDIDATES_631A[
            dataset_631a
        ]
    )

    print(
        f"\n--- {dataset_631a} FINAL AUTHORITY CANDIDATES ---"
    )

    print(
        "Candidate count                  :",
        len(
            candidates_631a
        ),
    )

    if not candidates_631a:
        print(
            "  NONE FOUND BY CURRENT LOCATOR"
        )
        continue

    for idx_631a, row_631a in enumerate(
        candidates_631a,
        start=1,
    ):

        print(
            f"\n{idx_631a:02d}. {row_631a['name']}"
        )

        print(
            "    score                         :",
            row_631a[
                "score"
            ],
        )

        print(
            "    top-level keys                :",
            row_631a[
                "top_level_keys"
            ],
        )

        if row_631a[
            "selected_fields"
        ]:
            print(
                "    selected reporting fields:"
            )

            for (
                path_631a,
                value_631a,
            ) in row_631a[
                "selected_fields"
            ]:
                print(
                    f"      {path_631a} = {value_631a}"
                )
        else:
            print(
                "    selected reporting fields     : NONE"
            )


print("\n--- LOCATOR SUMMARY ---")

print(
    "Candidate counts                 :",
    dataset_candidate_counts_631a,
)

print(
    "Datasets without candidates       :",
    datasets_without_candidates_631a,
)

print(
    "Locator audit SHA-256             :",
    FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_SHA256_631A,
)


print(
    "\nPASS — final resident reporting authorities were located "
    "without performing any scientific recomputation."
)

print(
    "Next: bind one authoritative final record per dataset and construct "
    "the FD001–FD004 cross-dataset evidence table from frozen results only."
)

In [ ]:
# STEP 6.31B — FD001 / FD002 FINAL AUTHORITY RECOVERY
#
# Purpose:
# Recover trustworthy FINAL reporting evidence for FD001 and FD002 after
# STEP 6.31A found no obvious resident final-authority object names.
#
# Evidence sources inspected:
# 1) resident FD001/FD002 globals, regardless of object naming
# 2) notebook input history (In)
# 3) existing JSON checkpoints under the established checkpoints directory
#
# REPORTING / STATIC RECOVERY ONLY.
#
# NO model inference.
# NO MC dropout.
# NO corruption execution.
# NO threshold/policy selection.
# NO scientific metric recomputation from raw predictions/labels.
# NO TEST/RUL file access.

import os
import re
import json
import hashlib
import numbers


# =============================================================================
# 0) PRECONDITION
# =============================================================================

required_631b = (
    "all_checks_passed_631a",
    "FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_631A",
    "FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_SHA256_631A",
)

missing_631b = [
    name
    for name in required_631b
    if name not in globals()
]

if missing_631b:
    raise RuntimeError(
        "Missing STEP 6.31B authority object(s): "
        + ", ".join(missing_631b)
    )

if not bool(all_checks_passed_631a):
    raise RuntimeError(
        "STEP 6.31A must remain passed."
    )


def _json_sha256_631b(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631b(
        FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_631A
    )
    !=
    str(
        FD004_CROSS_DATASET_AUTHORITY_LOCATOR_AUDIT_SHA256_631A
    )
):
    raise RuntimeError(
        "STEP 6.31A locator audit changed."
    )


# =============================================================================
# 1) REPORTING-FIELD EXTRACTION
# =============================================================================

report_field_re_631b = re.compile(
    r"status|"
    r"threshold|"
    r"coverage|"
    r"endpoint_count|"
    r"accepted_count|"
    r"abstained_count|"
    r"full_mae|"
    r"accepted_mae|"
    r"abstained_mae|"
    r"selective.*gain|"
    r"relative.*reduction|"
    r"positive_gain|"
    r"negative_gain|"
    r"scenario_count|"
    r"heldout|"
    r"test.*mae|"
    r"final.*mae|"
    r"closed",
    flags=re.IGNORECASE,
)


def _scalar_repr_631b(value):
    if isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        return repr(value)

    return None


def _extract_reporting_fields_631b(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 8:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():

            key_text = str(key)

            new_path = (
                path
                +
                (
                    key_text,
                )
            )

            path_text = ".".join(
                new_path
            )

            scalar = _scalar_repr_631b(
                child
            )

            if (
                scalar is not None
                and
                report_field_re_631b.search(
                    path_text
                )
            ):
                output.append(
                    (
                        path_text,
                        scalar,
                    )
                )

            _extract_reporting_fields_631b(
                child,
                new_path,
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) <= 100:
            for idx, child in enumerate(
                value
            ):
                _extract_reporting_fields_631b(
                    child,
                    path
                    +
                    (
                        f"[{idx}]",
                    ),
                    depth + 1,
                    output,
                )

    return output


def _evidence_score_631b(
    name,
    fields,
):
    text = (
        name
        +
        " "
        +
        " ".join(
            path
            +
            " "
            +
            value
            for path, value
            in fields
        )
    ).lower()

    score = 0

    weighted_tokens = (
        ("final", 40),
        ("closure", 40),
        ("conclusion", 35),
        ("heldout", 35),
        ("test", 20),
        ("full_mae", 25),
        ("accepted_mae", 25),
        ("abstained_mae", 25),
        ("selective_mae_gain", 25),
        ("relative", 15),
        ("coverage", 15),
        ("threshold", 10),
        ("endpoint_count", 10),
        ("complete_reporting_only", 35),
        ("closed", 20),
    )

    for token, weight in weighted_tokens:
        if token in text:
            score += weight

    return int(score)


# =============================================================================
# 2) SEARCH ALL RESIDENT FD001 / FD002 GLOBAL DICTS
# =============================================================================

resident_candidates_631b = {
    "FD001": [],
    "FD002": [],
}

for name_631b, value_631b in list(
    globals().items()
):

    if not isinstance(
        value_631b,
        dict,
    ):
        continue

    upper_name_631b = (
        name_631b.upper()
    )

    dataset_631b = None

    if "FD001" in upper_name_631b:
        dataset_631b = "FD001"

    elif "FD002" in upper_name_631b:
        dataset_631b = "FD002"

    if dataset_631b is None:
        continue


    fields_631b = (
        _extract_reporting_fields_631b(
            value_631b
        )
    )

    if not fields_631b:
        continue


    resident_candidates_631b[
        dataset_631b
    ].append(
        {
            "name":
                name_631b,

            "score":
                _evidence_score_631b(
                    name_631b,
                    fields_631b,
                ),

            "top_level_keys":
                tuple(
                    str(k)
                    for k in value_631b.keys()
                ),

            "fields":
                tuple(
                    fields_631b[
                        :80
                    ]
                ),
        }
    )


for dataset_631b in (
    "FD001",
    "FD002",
):
    resident_candidates_631b[
        dataset_631b
    ] = sorted(
        resident_candidates_631b[
            dataset_631b
        ],
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "name"
            ],
        ),
    )


# =============================================================================
# 3) SEARCH NOTEBOOK HISTORY FOR FINAL FD001 / FD002 REPORTING CELLS
# =============================================================================

history_candidates_631b = {
    "FD001": [],
    "FD002": [],
}


history_signal_re_631b = re.compile(
    r"full_mae|"
    r"accepted_mae|"
    r"abstained_mae|"
    r"selective_mae_gain|"
    r"relative_mae|"
    r"realized_coverage|"
    r"final.*closure|"
    r"final.*conclusion|"
    r"heldout.*test|"
    r"complete_reporting_only",
    flags=re.IGNORECASE,
)


if "In" in globals():

    try:
        for sequence_631b, text_631b in enumerate(
            In
        ):

            if not isinstance(
                text_631b,
                str,
            ):
                continue

            lower_631b = (
                text_631b.lower()
            )

            for dataset_631b in (
                "FD001",
                "FD002",
            ):

                if dataset_631b.lower() not in lower_631b:
                    continue

                if not history_signal_re_631b.search(
                    text_631b
                ):
                    continue


                score_631b = 0

                for token, weight in (
                    ("final", 30),
                    ("closure", 30),
                    ("heldout", 30),
                    ("full_mae", 20),
                    ("accepted_mae", 20),
                    ("abstained_mae", 20),
                    ("selective_mae_gain", 25),
                    ("relative_mae", 20),
                    ("complete_reporting_only", 30),
                ):
                    if token in lower_631b:
                        score_631b += weight


                lines_631b = (
                    text_631b.splitlines()
                )

                match_lines_631b = [
                    idx
                    for idx, line
                    in enumerate(
                        lines_631b,
                        start=1,
                    )
                    if history_signal_re_631b.search(
                        line
                    )
                ]


                snippets_631b = []

                used_ranges_631b = []

                for match_line_631b in match_lines_631b:

                    start_631b = max(
                        1,
                        match_line_631b - 5,
                    )

                    end_631b = min(
                        len(
                            lines_631b
                        ),
                        match_line_631b + 8,
                    )

                    if any(
                        not (
                            end_631b < used_start
                            or
                            start_631b > used_end
                        )
                        for used_start, used_end
                        in used_ranges_631b
                    ):
                        continue

                    used_ranges_631b.append(
                        (
                            start_631b,
                            end_631b,
                        )
                    )

                    snippets_631b.append(
                        tuple(
                            (
                                line_no_631b,
                                lines_631b[
                                    line_no_631b - 1
                                ],
                            )
                            for line_no_631b
                            in range(
                                start_631b,
                                end_631b + 1,
                            )
                        )
                    )

                    if len(
                        snippets_631b
                    ) >= 6:
                        break


                history_candidates_631b[
                    dataset_631b
                ].append(
                    {
                        "sequence":
                            int(
                                sequence_631b
                            ),

                        "score":
                            int(
                                score_631b
                            ),

                        "sha256":
                            hashlib.sha256(
                                text_631b.encode(
                                    "utf-8"
                                )
                            ).hexdigest(),

                        "snippets":
                            tuple(
                                snippets_631b
                            ),
                    }
                )

    except Exception:
        pass


for dataset_631b in (
    "FD001",
    "FD002",
):
    history_candidates_631b[
        dataset_631b
    ] = sorted(
        history_candidates_631b[
            dataset_631b
        ],
        key=lambda row: (
            -row[
                "score"
            ],
            -row[
                "sequence"
            ],
        ),
    )


# =============================================================================
# 4) SEARCH EXISTING JSON CHECKPOINTS
#
# This reads reporting/checkpoint JSON only.
# It does NOT touch raw C-MAPSS TEST/RUL files.
# =============================================================================

checkpoint_dir_631b = (
    r"C:\RUL_Reliability_Research\checkpoints"
)

checkpoint_candidates_631b = {
    "FD001": [],
    "FD002": [],
}


if os.path.isdir(
    checkpoint_dir_631b
):

    for filename_631b in os.listdir(
        checkpoint_dir_631b
    ):

        lower_filename_631b = (
            filename_631b.lower()
        )

        if not lower_filename_631b.endswith(
            ".json"
        ):
            continue


        dataset_631b = None

        if "fd001" in lower_filename_631b:
            dataset_631b = "FD001"

        elif "fd002" in lower_filename_631b:
            dataset_631b = "FD002"

        if dataset_631b is None:
            continue


        path_631b = os.path.join(
            checkpoint_dir_631b,
            filename_631b,
        )

        try:
            with open(
                path_631b,
                "r",
                encoding="utf-8",
            ) as handle_631b:

                payload_631b = json.load(
                    handle_631b
                )

        except Exception as exc_631b:

            checkpoint_candidates_631b[
                dataset_631b
            ].append(
                {
                    "filename":
                        filename_631b,

                    "path":
                        path_631b,

                    "load_error":
                        repr(
                            exc_631b
                        ),

                    "score":
                        -1,

                    "fields":
                        (),
                }
            )

            continue


        fields_631b = (
            _extract_reporting_fields_631b(
                payload_631b
            )
        )


        score_631b = (
            _evidence_score_631b(
                filename_631b,
                fields_631b,
            )
        )


        with open(
            path_631b,
            "rb",
        ) as handle_631b:
            file_sha_631b = hashlib.sha256(
                handle_631b.read()
            ).hexdigest()


        checkpoint_candidates_631b[
            dataset_631b
        ].append(
            {
                "filename":
                    filename_631b,

                "path":
                    path_631b,

                "file_sha256":
                    file_sha_631b,

                "score":
                    int(
                        score_631b
                    ),

                "top_level_keys":
                    tuple(
                        str(k)
                        for k in (
                            payload_631b.keys()
                            if isinstance(
                                payload_631b,
                                dict,
                            )
                            else
                            ()
                        )
                    ),

                "fields":
                    tuple(
                        fields_631b[
                            :100
                        ]
                    ),
            }
        )


for dataset_631b in (
    "FD001",
    "FD002",
):
    checkpoint_candidates_631b[
        dataset_631b
    ] = sorted(
        checkpoint_candidates_631b[
            dataset_631b
        ],
        key=lambda row: (
            -row.get(
                "score",
                -1,
            ),
            row[
                "filename"
            ],
        ),
    )


# =============================================================================
# 5) FREEZE RECOVERY AUDIT
# =============================================================================

FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_631B = {
    "FD001": {
        "resident_candidates":
            tuple(
                resident_candidates_631b[
                    "FD001"
                ][
                    :15
                ]
            ),

        "history_candidates":
            tuple(
                history_candidates_631b[
                    "FD001"
                ][
                    :12
                ]
            ),

        "checkpoint_candidates":
            tuple(
                checkpoint_candidates_631b[
                    "FD001"
                ][
                    :15
                ]
            ),
    },

    "FD002": {
        "resident_candidates":
            tuple(
                resident_candidates_631b[
                    "FD002"
                ][
                    :15
                ]
            ),

        "history_candidates":
            tuple(
                history_candidates_631b[
                    "FD002"
                ][
                    :12
                ]
            ),

        "checkpoint_candidates":
            tuple(
                checkpoint_candidates_631b[
                    "FD002"
                ][
                    :15
                ]
            ),
    },
}


audit_payload_631b = {
    "stage":
        "6.31B",

    "status":
        "FD001_FD002_FINAL_REPORTING_AUTHORITY_RECOVERY_COMPLETE",

    "fd001_resident_candidate_count":
        len(
            resident_candidates_631b[
                "FD001"
            ]
        ),

    "fd001_history_candidate_count":
        len(
            history_candidates_631b[
                "FD001"
            ]
        ),

    "fd001_checkpoint_candidate_count":
        len(
            checkpoint_candidates_631b[
                "FD001"
            ]
        ),

    "fd002_resident_candidate_count":
        len(
            resident_candidates_631b[
                "FD002"
            ]
        ),

    "fd002_history_candidate_count":
        len(
            history_candidates_631b[
                "FD002"
            ]
        ),

    "fd002_checkpoint_candidate_count":
        len(
            checkpoint_candidates_631b[
                "FD002"
            ]
        ),

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "corruption_execution":
        False,

    "threshold_selection":
        False,

    "policy_selection":
        False,

    "raw_test_or_rul_access":
        False,

    "reporting_only":
        True,
}


FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_631B = (
    audit_payload_631b
)

FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_SHA256_631B = (
    _json_sha256_631b(
        audit_payload_631b
    )
)

all_checks_passed_631b = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31B — FD001 / FD002 FINAL AUTHORITY RECOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Corruption execution              :", False)
print("Threshold/policy selection        :", False)
print("Raw TEST/RUL access               :", False)
print("Reporting-only recovery           :", True)


for dataset_631b in (
    "FD001",
    "FD002",
):

    print(
        f"\n{'=' * 72}"
    )

    print(
        f"{dataset_631b} RECOVERY"
    )

    print(
        f"{'=' * 72}"
    )


    print(
        "\n--- RESIDENT GLOBAL CANDIDATES ---"
    )

    resident_rows_631b = (
        resident_candidates_631b[
            dataset_631b
        ]
    )

    print(
        "Candidate count                  :",
        len(
            resident_rows_631b
        ),
    )

    for idx_631b, row_631b in enumerate(
        resident_rows_631b[
            :10
        ],
        start=1,
    ):

        print(
            f"\n{idx_631b:02d}. "
            f"{row_631b['name']} "
            f"(score={row_631b['score']})"
        )

        for path_631b, value_631b in row_631b[
            "fields"
        ][
            :35
        ]:
            print(
                f"      {path_631b} = {value_631b}"
            )


    print(
        "\n--- NOTEBOOK HISTORY CANDIDATES ---"
    )

    history_rows_631b = (
        history_candidates_631b[
            dataset_631b
        ]
    )

    print(
        "Candidate count                  :",
        len(
            history_rows_631b
        ),
    )

    for idx_631b, row_631b in enumerate(
        history_rows_631b[
            :6
        ],
        start=1,
    ):

        print(
            f"\nHISTORY {idx_631b}"
        )

        print(
            "  sequence                        :",
            row_631b[
                "sequence"
            ],
        )

        print(
            "  score                           :",
            row_631b[
                "score"
            ],
        )

        print(
            "  SHA-256                         :",
            row_631b[
                "sha256"
            ],
        )

        for snippet_631b in row_631b[
            "snippets"
        ][
            :5
        ]:

            print()

            for (
                line_no_631b,
                line_631b,
            ) in snippet_631b:

                print(
                    f"    "
                    f"{line_no_631b:05d} | "
                    f"{line_631b}"
                )


    print(
        "\n--- CHECKPOINT JSON CANDIDATES ---"
    )

    checkpoint_rows_631b = (
        checkpoint_candidates_631b[
            dataset_631b
        ]
    )

    print(
        "Candidate count                  :",
        len(
            checkpoint_rows_631b
        ),
    )

    for idx_631b, row_631b in enumerate(
        checkpoint_rows_631b[
            :10
        ],
        start=1,
    ):

        print(
            f"\nCHECKPOINT {idx_631b}"
        )

        print(
            "  filename                        :",
            row_631b[
                "filename"
            ],
        )

        print(
            "  score                           :",
            row_631b.get(
                "score"
            ),
        )

        if "load_error" in row_631b:
            print(
                "  load error                      :",
                row_631b[
                    "load_error"
                ],
            )
            continue

        print(
            "  file SHA-256                    :",
            row_631b[
                "file_sha256"
            ],
        )

        for path_631b, value_631b in row_631b[
            "fields"
        ][
            :40
        ]:
            print(
                f"      {path_631b} = {value_631b}"
            )


print(
    "\n--- RECOVERY AUDIT ---"
)

print(
    "FD001 resident/history/checkpoint :",
    (
        audit_payload_631b[
            "fd001_resident_candidate_count"
        ],
        audit_payload_631b[
            "fd001_history_candidate_count"
        ],
        audit_payload_631b[
            "fd001_checkpoint_candidate_count"
        ],
    ),
)

print(
    "FD002 resident/history/checkpoint :",
    (
        audit_payload_631b[
            "fd002_resident_candidate_count"
        ],
        audit_payload_631b[
            "fd002_history_candidate_count"
        ],
        audit_payload_631b[
            "fd002_checkpoint_candidate_count"
        ],
    ),
)

print(
    "Recovery audit SHA-256            :",
    FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_SHA256_631B,
)

print(
    "\nPASS — FD001/FD002 final reporting evidence was searched "
    "without scientific recomputation."
)

print(
    "Next: select and bind the strongest frozen authority for FD001 "
    "and FD002, then build the four-dataset synthesis table."
)

In [ ]:
# STEP 6.31C — FD001 / FD002 DEEP HISTORICAL FINAL-EVIDENCE RECOVERY
#
# Why this step exists:
# 6.31A / 6.31B found no trustworthy named resident final authority for
# FD001 or FD002. The apparent notebook-history hits were recent locator
# cells, not the historical result cells.
#
# This step searches more deeply:
#   1) ALL resident dict/list/tuple globals, regardless of variable name
#   2) ALL resident source-text strings, regardless of variable name
#   3) older notebook input cells, excluding recent 6.30W/6.31 work
#   4) text/JSON/notebook files recursively under the research project,
#      while explicitly excluding raw dataset directories
#
# REPORTING EVIDENCE RECOVERY ONLY.
#
# NO model inference.
# NO MC dropout.
# NO corruption execution.
# NO metric recomputation from predictions/labels.
# NO raw TEST/RUL file access.
# NO threshold/policy selection.

import os
import re
import json
import hashlib
import numbers


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631c = (
    "all_checks_passed_631b",
    "FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_631B",
    "FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_SHA256_631B",
)

missing_631c = [
    name
    for name in required_631c
    if name not in globals()
]

if missing_631c:
    raise RuntimeError(
        "Missing STEP 6.31C authority object(s): "
        + ", ".join(missing_631c)
    )

if not bool(all_checks_passed_631b):
    raise RuntimeError(
        "STEP 6.31B must remain passed."
    )


def _json_sha256_631c(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631c(
        FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_631B
    )
    !=
    str(
        FD004_HISTORICAL_FINAL_AUTHORITY_RECOVERY_AUDIT_SHA256_631B
    )
):
    raise RuntimeError(
        "STEP 6.31B recovery audit changed."
    )


# =============================================================================
# 1) SEARCH DEFINITIONS
# =============================================================================

datasets_631c = (
    "FD001",
    "FD002",
)


report_signal_re_631c = re.compile(
    r"full[_ ]?(?:test[_ ]?)?mae|"
    r"accepted[_ ]?mae|"
    r"abstained[_ ]?mae|"
    r"selective[_ ]?(?:mae[_ ]?)?gain|"
    r"relative[_ ]?(?:mae[_ ]?)?reduction|"
    r"realized[_ ]?coverage|"
    r"heldout|"
    r"held-out|"
    r"endpoint_count|"
    r"accepted_count|"
    r"abstained_count|"
    r"frozen[_ ]?risk[_ ]?threshold|"
    r"complete_reporting_only|"
    r"final[_ ]?(?:dataset[_ ]?)?closure|"
    r"final[_ ]?scientific[_ ]?conclusion",
    flags=re.IGNORECASE,
)


high_value_re_631c = re.compile(
    r"full[_ ]?(?:test[_ ]?)?mae|"
    r"accepted[_ ]?mae|"
    r"abstained[_ ]?mae|"
    r"selective[_ ]?(?:mae[_ ]?)?gain|"
    r"relative[_ ]?(?:mae[_ ]?)?reduction|"
    r"complete_reporting_only_evidence_locked|"
    r"final[_ ]?dataset[_ ]?closure|"
    r"final[_ ]?scientific[_ ]?conclusion",
    flags=re.IGNORECASE,
)


numeric_metric_re_631c = re.compile(
    r"(?:full[_ ]?(?:test[_ ]?)?mae|"
    r"accepted[_ ]?mae|"
    r"abstained[_ ]?mae|"
    r"selective[_ ]?(?:mae[_ ]?)?gain|"
    r"relative[_ ]?(?:mae[_ ]?)?reduction|"
    r"realized[_ ]?coverage)"
    r".{0,100}?"
    r"(?:=|:)"
    r".{0,20}?"
    r"[-+]?(?:\d+\.\d+|\d+)",
    flags=re.IGNORECASE,
)


def _is_recent_locator_text_631c(text):
    lower = text.lower()

    return bool(
        "step 6.30w-" in lower
        or
        "step 6.31" in lower
        or
        "630w10" in lower
        or
        "631a" in lower
        or
        "631b" in lower
        or
        "631c" in lower
    )


def _text_score_631c(
    dataset,
    text,
):
    lower = text.lower()

    if dataset.lower() not in lower:
        return -1

    score = 0

    weighted_tokens = (
        ("complete_reporting_only_evidence_locked", 100),
        ("final_dataset_closure", 80),
        ("final scientific conclusion", 70),
        ("final_scientific_conclusion", 70),
        ("heldout_test", 55),
        ("held-out test", 55),
        ("full_mae", 45),
        ("full mae", 45),
        ("accepted_mae", 45),
        ("accepted mae", 45),
        ("abstained_mae", 45),
        ("abstained mae", 45),
        ("selective_mae_gain", 50),
        ("selective mae gain", 50),
        ("relative_mae_reduction", 45),
        ("relative mae reduction", 45),
        ("realized_coverage", 30),
        ("realized coverage", 30),
        ("endpoint_count", 20),
        ("frozen_risk_threshold", 20),
        ("accepted_count", 15),
        ("abstained_count", 15),
        ("reporting_only", 25),
        ("closed", 15),
    )

    for token, weight in weighted_tokens:
        if token in lower:
            score += weight

    numeric_hits = len(
        numeric_metric_re_631c.findall(
            text
        )
    )

    score += min(
        numeric_hits,
        12,
    ) * 25

    return int(score)


# =============================================================================
# 2) RECURSIVE STRUCTURED-OBJECT SEARCH
#
# Unlike 6.31B, the GLOBAL NAME does not need to contain FD001/FD002.
# We inspect object CONTENT.
# =============================================================================

structured_candidates_631c = {
    "FD001": [],
    "FD002": [],
}


def _scalar_repr_631c(value):
    if isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        return repr(value)

    return None


def _extract_structured_fields_631c(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 8:
        return output

    if isinstance(
        value,
        dict,
    ):

        for key, child in value.items():

            new_path = (
                path
                +
                (
                    str(key),
                )
            )

            path_text = ".".join(
                new_path
            )

            scalar = _scalar_repr_631c(
                child
            )

            if (
                scalar is not None
                and
                report_signal_re_631c.search(
                    path_text
                    +
                    " "
                    +
                    scalar
                )
            ):
                output.append(
                    (
                        path_text,
                        scalar,
                    )
                )

            _extract_structured_fields_631c(
                child,
                new_path,
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        if len(value) <= 200:

            for idx, child in enumerate(
                value
            ):

                _extract_structured_fields_631c(
                    child,
                    path
                    +
                    (
                        f"[{idx}]",
                    ),
                    depth + 1,
                    output,
                )

    return output


for name_631c, value_631c in list(
    globals().items()
):

    if not isinstance(
        value_631c,
        (
            dict,
            list,
            tuple,
        ),
    ):
        continue

    try:
        serialized_631c = json.dumps(
            value_631c,
            default=str,
        )
    except Exception:
        continue

    lower_serialized_631c = (
        serialized_631c.lower()
    )

    for dataset_631c in datasets_631c:

        if dataset_631c.lower() not in lower_serialized_631c:
            continue

        if not report_signal_re_631c.search(
            serialized_631c
        ):
            continue

        fields_631c = _extract_structured_fields_631c(
            value_631c
        )

        score_631c = _text_score_631c(
            dataset_631c,
            (
                name_631c
                +
                "\n"
                +
                serialized_631c
            ),
        )

        structured_candidates_631c[
            dataset_631c
        ].append(
            {
                "global_name":
                    name_631c,

                "type":
                    type(
                        value_631c
                    ).__name__,

                "score":
                    score_631c,

                "fields":
                    tuple(
                        fields_631c[
                            :100
                        ]
                    ),
            }
        )


for dataset_631c in datasets_631c:

    structured_candidates_631c[
        dataset_631c
    ] = sorted(
        structured_candidates_631c[
            dataset_631c
        ],
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "global_name"
            ],
        ),
    )


# =============================================================================
# 3) ALL RESIDENT SOURCE-TEXT GLOBALS
#
# No requirement that the variable name itself mentions FD001/FD002.
# =============================================================================

source_text_candidates_631c = {
    "FD001": [],
    "FD002": [],
}

seen_text_sha_631c = set()


def _add_text_candidate_631c(
    dataset,
    origin,
    text,
):
    if not isinstance(
        text,
        str,
    ):
        return

    if len(text) < 100:
        return

    if _is_recent_locator_text_631c(
        text
    ):
        return

    if dataset.lower() not in text.lower():
        return

    if not report_signal_re_631c.search(
        text
    ):
        return

    sha = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()

    key = (
        dataset,
        sha,
    )

    if key in seen_text_sha_631c:
        return

    seen_text_sha_631c.add(
        key
    )

    source_text_candidates_631c[
        dataset
    ].append(
        {
            "origin":
                origin,

            "sha256":
                sha,

            "score":
                _text_score_631c(
                    dataset,
                    text,
                ),

            "text":
                text,
        }
    )


for name_631c, value_631c in list(
    globals().items()
):

    if not isinstance(
        value_631c,
        str,
    ):
        continue

    for dataset_631c in datasets_631c:

        _add_text_candidate_631c(
            dataset_631c,
            "global:" + name_631c,
            value_631c,
        )


# =============================================================================
# 4) OLDER NOTEBOOK INPUT HISTORY
# =============================================================================

if "In" in globals():

    try:

        for sequence_631c, text_631c in enumerate(
            In
        ):

            if not isinstance(
                text_631c,
                str,
            ):
                continue

            if _is_recent_locator_text_631c(
                text_631c
            ):
                continue

            for dataset_631c in datasets_631c:

                _add_text_candidate_631c(
                    dataset_631c,
                    f"In[{sequence_631c}]",
                    text_631c,
                )

    except Exception:
        pass


for dataset_631c in datasets_631c:

    source_text_candidates_631c[
        dataset_631c
    ] = sorted(
        source_text_candidates_631c[
            dataset_631c
        ],
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "origin"
            ],
        ),
    )


# =============================================================================
# 5) PROJECT FILE SEARCH
#
# Search text-like research artifacts recursively.
# Explicitly DO NOT enter raw dataset directories.
# =============================================================================

project_root_631c = (
    r"C:\RUL_Reliability_Research"
)

allowed_extensions_631c = {
    ".json",
    ".txt",
    ".py",
    ".md",
    ".log",
    ".yaml",
    ".yml",
    ".ipynb",
}

excluded_dir_tokens_631c = (
    "02_dataset",
    "__pycache__",
    ".git",
    ".venv",
    "venv",
    "site-packages",
)


file_candidates_631c = {
    "FD001": [],
    "FD002": [],
}


def _allowed_path_631c(path):
    lower = path.lower().replace(
        "/",
        "\\",
    )

    return not any(
        (
            "\\" + token.lower() + "\\"
            in lower
            or
            lower.endswith(
                "\\" + token.lower()
            )
        )
        for token
        in excluded_dir_tokens_631c
    )


def _inspect_text_file_631c(
    path,
):
    try:
        size = os.path.getsize(
            path
        )
    except OSError:
        return None

    # Avoid enormous unrelated artifacts.
    if size > 20_000_000:
        return None

    try:
        with open(
            path,
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as handle:
            return handle.read()
    except Exception:
        return None


if os.path.isdir(
    project_root_631c
):

    for root_631c, dirs_631c, files_631c in os.walk(
        project_root_631c
    ):

        dirs_631c[:] = [
            d
            for d in dirs_631c
            if _allowed_path_631c(
                os.path.join(
                    root_631c,
                    d,
                )
            )
        ]

        if not _allowed_path_631c(
            root_631c
        ):
            continue

        for filename_631c in files_631c:

            extension_631c = os.path.splitext(
                filename_631c
            )[
                1
            ].lower()

            if extension_631c not in allowed_extensions_631c:
                continue

            path_631c = os.path.join(
                root_631c,
                filename_631c,
            )

            if not _allowed_path_631c(
                path_631c
            ):
                continue

            text_631c = _inspect_text_file_631c(
                path_631c
            )

            if not text_631c:
                continue

            # Ignore files consisting only of our recent locator work.
            # We still allow older notebooks/files that contain older evidence.
            lower_text_631c = text_631c.lower()

            for dataset_631c in datasets_631c:

                if dataset_631c.lower() not in lower_text_631c:
                    continue

                if not report_signal_re_631c.search(
                    text_631c
                ):
                    continue

                score_631c = _text_score_631c(
                    dataset_631c,
                    text_631c,
                )

                if score_631c <= 0:
                    continue

                try:
                    with open(
                        path_631c,
                        "rb",
                    ) as handle:
                        file_sha_631c = hashlib.sha256(
                            handle.read()
                        ).hexdigest()
                except Exception:
                    file_sha_631c = None

                file_candidates_631c[
                    dataset_631c
                ].append(
                    {
                        "path":
                            path_631c,

                        "file_sha256":
                            file_sha_631c,

                        "score":
                            score_631c,

                        "text":
                            text_631c,
                    }
                )


for dataset_631c in datasets_631c:

    file_candidates_631c[
        dataset_631c
    ] = sorted(
        file_candidates_631c[
            dataset_631c
        ],
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "path"
            ],
        ),
    )


# =============================================================================
# 6) CONTEXT EXTRACTION
# =============================================================================

context_re_631c = re.compile(
    r"FD00[12]|"
    r"full[_ ]?(?:test[_ ]?)?mae|"
    r"accepted[_ ]?mae|"
    r"abstained[_ ]?mae|"
    r"selective[_ ]?(?:mae[_ ]?)?gain|"
    r"relative[_ ]?(?:mae[_ ]?)?reduction|"
    r"realized[_ ]?coverage|"
    r"complete_reporting_only|"
    r"final[_ ]?dataset[_ ]?closure|"
    r"final[_ ]?scientific[_ ]?conclusion",
    flags=re.IGNORECASE,
)


def _extract_contexts_631c(
    dataset,
    text,
    max_contexts=8,
):
    lines = text.splitlines()

    matches = []

    for idx, line in enumerate(
        lines,
        start=1,
    ):
        if not context_re_631c.search(
            line
        ):
            continue

        # Prefer lines directly connected to the desired dataset or metrics.
        local_start = max(
            0,
            idx - 4,
        )

        local_end = min(
            len(lines),
            idx + 4,
        )

        local_text = "\n".join(
            lines[
                local_start:
                local_end
            ]
        )

        if dataset.lower() not in local_text.lower():
            continue

        matches.append(
            idx
        )

    contexts = []
    occupied = []

    for match_line in matches:

        start = max(
            1,
            match_line - 7,
        )

        end = min(
            len(lines),
            match_line + 12,
        )

        if any(
            not (
                end < used_start
                or
                start > used_end
            )
            for used_start, used_end
            in occupied
        ):
            continue

        occupied.append(
            (
                start,
                end,
            )
        )

        contexts.append(
            tuple(
                (
                    line_no,
                    lines[
                        line_no - 1
                    ],
                )
                for line_no
                in range(
                    start,
                    end + 1,
                )
            )
        )

        if len(contexts) >= max_contexts:
            break

    return tuple(
        contexts
    )


# Create compact display objects without retaining duplicate giant source text.
display_source_candidates_631c = {
    "FD001": [],
    "FD002": [],
}

display_file_candidates_631c = {
    "FD001": [],
    "FD002": [],
}


for dataset_631c in datasets_631c:

    for row_631c in source_text_candidates_631c[
        dataset_631c
    ][
        :15
    ]:

        display_source_candidates_631c[
            dataset_631c
        ].append(
            {
                "origin":
                    row_631c[
                        "origin"
                    ],

                "sha256":
                    row_631c[
                        "sha256"
                    ],

                "score":
                    row_631c[
                        "score"
                    ],

                "contexts":
                    _extract_contexts_631c(
                        dataset_631c,
                        row_631c[
                            "text"
                        ],
                    ),
            }
        )


    for row_631c in file_candidates_631c[
        dataset_631c
    ][
        :15
    ]:

        display_file_candidates_631c[
            dataset_631c
        ].append(
            {
                "path":
                    row_631c[
                        "path"
                    ],

                "file_sha256":
                    row_631c[
                        "file_sha256"
                    ],

                "score":
                    row_631c[
                        "score"
                    ],

                "contexts":
                    _extract_contexts_631c(
                        dataset_631c,
                        row_631c[
                            "text"
                        ],
                    ),
            }
        )


# =============================================================================
# 7) FREEZE RECOVERY AUDIT
# =============================================================================

FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_631C = {
    "FD001": {
        "structured_candidates":
            tuple(
                structured_candidates_631c[
                    "FD001"
                ][
                    :20
                ]
            ),

        "source_candidates":
            tuple(
                display_source_candidates_631c[
                    "FD001"
                ]
            ),

        "file_candidates":
            tuple(
                display_file_candidates_631c[
                    "FD001"
                ]
            ),
    },

    "FD002": {
        "structured_candidates":
            tuple(
                structured_candidates_631c[
                    "FD002"
                ][
                    :20
                ]
            ),

        "source_candidates":
            tuple(
                display_source_candidates_631c[
                    "FD002"
                ]
            ),

        "file_candidates":
            tuple(
                display_file_candidates_631c[
                    "FD002"
                ]
            ),
    },
}


audit_payload_631c = {
    "stage":
        "6.31C",

    "status":
        "FD001_FD002_DEEP_HISTORICAL_FINAL_EVIDENCE_RECOVERY_COMPLETE",

    "fd001_structured_candidate_count":
        len(
            structured_candidates_631c[
                "FD001"
            ]
        ),

    "fd001_source_candidate_count":
        len(
            source_text_candidates_631c[
                "FD001"
            ]
        ),

    "fd001_file_candidate_count":
        len(
            file_candidates_631c[
                "FD001"
            ]
        ),

    "fd002_structured_candidate_count":
        len(
            structured_candidates_631c[
                "FD002"
            ]
        ),

    "fd002_source_candidate_count":
        len(
            source_text_candidates_631c[
                "FD002"
            ]
        ),

    "fd002_file_candidate_count":
        len(
            file_candidates_631c[
                "FD002"
            ]
        ),

    "raw_dataset_directory_excluded":
        True,

    "raw_test_rul_access":
        False,

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "corruption_execution":
        False,

    "threshold_policy_selection":
        False,

    "reporting_only":
        True,
}


FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_631C = (
    audit_payload_631c
)

FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_SHA256_631C = (
    _json_sha256_631c(
        audit_payload_631c
    )
)

all_checks_passed_631c = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31C — FD001 / FD002 DEEP HISTORICAL "
    "FINAL-EVIDENCE RECOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Corruption execution              :", False)
print("Threshold/policy selection        :", False)
print("Raw dataset directory searched    :", False)
print("Raw TEST/RUL access               :", False)
print("Reporting-only recovery           :", True)


for dataset_631c in datasets_631c:

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        dataset_631c
    )

    print(
        "=" * 80
    )


    # -------------------------------------------------------------------------
    # STRUCTURED OBJECTS
    # -------------------------------------------------------------------------

    print(
        "\n--- CONTENT-MATCHED RESIDENT STRUCTURED OBJECTS ---"
    )

    structured_rows_631c = (
        structured_candidates_631c[
            dataset_631c
        ]
    )

    print(
        "Candidate count                  :",
        len(
            structured_rows_631c
        ),
    )

    for idx, row in enumerate(
        structured_rows_631c[
            :12
        ],
        start=1,
    ):

        print(
            f"\nSTRUCTURED {idx}: "
            f"{row['global_name']}"
        )

        print(
            "  type                            :",
            row[
                "type"
            ],
        )

        print(
            "  score                           :",
            row[
                "score"
            ],
        )

        for path, value in row[
            "fields"
        ][
            :40
        ]:
            print(
                f"      {path} = {value}"
            )


    # -------------------------------------------------------------------------
    # SOURCE STRINGS / OLD INPUT CELLS
    # -------------------------------------------------------------------------

    print(
        "\n--- HISTORICAL SOURCE-TEXT EVIDENCE ---"
    )

    source_rows_631c = (
        display_source_candidates_631c[
            dataset_631c
        ]
    )

    print(
        "Candidate count                  :",
        len(
            source_text_candidates_631c[
                dataset_631c
            ]
        ),
    )

    print(
        "Displaying                       :",
        len(
            source_rows_631c
        ),
    )

    for idx, row in enumerate(
        source_rows_631c,
        start=1,
    ):

        print(
            f"\nSOURCE {idx}"
        )

        print(
            "  origin                          :",
            row[
                "origin"
            ],
        )

        print(
            "  score                           :",
            row[
                "score"
            ],
        )

        print(
            "  SHA-256                         :",
            row[
                "sha256"
            ],
        )

        for snippet in row[
            "contexts"
        ][
            :6
        ]:

            print()

            for line_no, line in snippet:

                print(
                    f"    "
                    f"{line_no:05d} | "
                    f"{line}"
                )


    # -------------------------------------------------------------------------
    # PROJECT FILES
    # -------------------------------------------------------------------------

    print(
        "\n--- PROJECT TEXT / JSON / NOTEBOOK EVIDENCE ---"
    )

    file_rows_631c = (
        display_file_candidates_631c[
            dataset_631c
        ]
    )

    print(
        "Candidate count                  :",
        len(
            file_candidates_631c[
                dataset_631c
            ]
        ),
    )

    print(
        "Displaying                       :",
        len(
            file_rows_631c
        ),
    )

    for idx, row in enumerate(
        file_rows_631c,
        start=1,
    ):

        print(
            f"\nFILE {idx}"
        )

        print(
            "  path                            :",
            row[
                "path"
            ],
        )

        print(
            "  score                           :",
            row[
                "score"
            ],
        )

        print(
            "  file SHA-256                    :",
            row[
                "file_sha256"
            ],
        )

        for snippet in row[
            "contexts"
        ][
            :6
        ]:

            print()

            for line_no, line in snippet:

                print(
                    f"    "
                    f"{line_no:05d} | "
                    f"{line}"
                )


print(
    "\n--- DEEP RECOVERY AUDIT ---"
)

print(
    "FD001 structured/source/files     :",
    (
        audit_payload_631c[
            "fd001_structured_candidate_count"
        ],
        audit_payload_631c[
            "fd001_source_candidate_count"
        ],
        audit_payload_631c[
            "fd001_file_candidate_count"
        ],
    ),
)

print(
    "FD002 structured/source/files     :",
    (
        audit_payload_631c[
            "fd002_structured_candidate_count"
        ],
        audit_payload_631c[
            "fd002_source_candidate_count"
        ],
        audit_payload_631c[
            "fd002_file_candidate_count"
        ],
    ),
)

print(
    "Raw dataset directory excluded    :",
    True,
)

print(
    "Deep recovery audit SHA-256       :",
    FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_SHA256_631C,
)

print(
    "\nPASS — deep historical reporting evidence was inspected "
    "without scientific recomputation or raw TEST/RUL access."
)

print(
    "Next: bind FD001/FD002 only if this recovery exposes genuine "
    "historical final-result evidence; otherwise leave unsupported "
    "cross-dataset fields explicitly unavailable rather than inventing them."
)

In [ ]:
# STEP 6.31D — FD001 / FD002 FINAL AUTHORITY + TARGET-CONVENTION DISCRIMINATOR
#
# Purpose:
# 1) Bind the genuine FD001 final evidence artifact recovered in 6.31C.
# 2) Recover ALL plausible FD002 final scoring/reporting summaries.
# 3) Identify which target convention each artifact reports:
#       - raw RUL
#       - capped / piecewise-capped RUL
#       - both
# 4) Prevent invalid cross-dataset comparison across mismatched targets.
#
# REPORTING-ONLY.
#
# NO model inference.
# NO MC dropout.
# NO new corruption.
# NO TEST/RUL raw-file access.
# NO metric recomputation from predictions/labels.
# NO threshold/policy changes.

import os
import re
import json
import csv
import hashlib
import numbers
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631d = (
    "all_checks_passed_631c",
    "FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_631C",
    "FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_SHA256_631C",
)

missing_631d = [
    name
    for name in required_631d
    if name not in globals()
]

if missing_631d:
    raise RuntimeError(
        "Missing STEP 6.31D authority object(s): "
        + ", ".join(missing_631d)
    )

if not bool(all_checks_passed_631c):
    raise RuntimeError(
        "STEP 6.31C must remain passed."
    )


def _json_sha256_631d(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631d(
        FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_631C
    )
    !=
    str(
        FD004_DEEP_HISTORICAL_FINAL_EVIDENCE_AUDIT_SHA256_631C
    )
):
    raise RuntimeError(
        "STEP 6.31C audit changed."
    )


# =============================================================================
# 1) EXACT FD001 FINAL EVIDENCE AUTHORITY
# =============================================================================

fd001_final_path_631d = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\fd001_final\fd001_final_evidence.json"
)

if not fd001_final_path_631d.is_file():
    raise RuntimeError(
        "Recovered FD001 final evidence file is missing: "
        + str(fd001_final_path_631d)
    )


with open(
    fd001_final_path_631d,
    "r",
    encoding="utf-8",
) as handle_631d:
    fd001_final_payload_631d = json.load(
        handle_631d
    )


with open(
    fd001_final_path_631d,
    "rb",
) as handle_631d:
    fd001_final_file_sha_631d = hashlib.sha256(
        handle_631d.read()
    ).hexdigest()


# =============================================================================
# 2) TARGET-CONVENTION CLASSIFIER
# =============================================================================

def _flatten_scalars_631d(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 10:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _flatten_scalars_631d(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (list, tuple),
    ):
        if len(value) <= 200:
            for idx, child in enumerate(value):
                _flatten_scalars_631d(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        output.append(
            (
                ".".join(path),
                value,
            )
        )

    return output


def _target_signature_631d(
    payload,
):
    scalars = _flatten_scalars_631d(
        payload
    )

    text = "\n".join(
        f"{path}={value}"
        for path, value
        in scalars
    ).lower()

    raw_signals = (
        "rul_raw",
        "raw_rul",
        "official_rul_raw",
        "uncapped",
        "raw rul",
    )

    capped_signals = (
        "rul_cap_125",
        "capped_125",
        "cap_125",
        "piecewise-capped",
        "piecewise capped",
        "capped rul",
    )

    has_raw = any(
        signal in text
        for signal in raw_signals
    )

    has_capped = any(
        signal in text
        for signal in capped_signals
    )

    if has_raw and has_capped:
        convention = "BOTH_RAW_AND_CAPPED_REFERENCED"
    elif has_raw:
        convention = "RAW_RUL_REFERENCED"
    elif has_capped:
        convention = "CAPPED_RUL_REFERENCED"
    else:
        convention = "TARGET_CONVENTION_NOT_EXPLICIT"

    return (
        convention,
        tuple(
            (
                path,
                value,
            )
            for path, value
            in scalars
            if any(
                token
                in (
                    path
                    +
                    " "
                    +
                    str(value)
                ).lower()
                for token
                in (
                    "target",
                    "rul_raw",
                    "raw_rul",
                    "official_rul",
                    "cap_125",
                    "capped",
                    "piecewise",
                )
            )
        ),
    )


(
    fd001_target_convention_631d,
    fd001_target_evidence_631d,
) = _target_signature_631d(
    fd001_final_payload_631d
)


# =============================================================================
# 3) EXTRACT REPORTING METRICS WITHOUT RECOMPUTATION
# =============================================================================

metric_re_631d = re.compile(
    r"coverage|"
    r"accepted.*count|"
    r"abstained.*count|"
    r"full.*mae|"
    r"accepted.*mae|"
    r"abstained.*mae|"
    r"selective.*gain|"
    r"relative.*reduction|"
    r"threshold|"
    r"test_engines|"
    r"endpoint_count|"
    r"rmse|"
    r"nasa.*score",
    flags=re.IGNORECASE,
)


def _extract_reporting_scalars_631d(
    payload,
):
    return tuple(
        (
            path,
            value,
        )
        for path, value
        in _flatten_scalars_631d(
            payload
        )
        if metric_re_631d.search(
            path
        )
    )


fd001_reporting_fields_631d = (
    _extract_reporting_scalars_631d(
        fd001_final_payload_631d
    )
)


# =============================================================================
# 4) TARGETED FD002 REPORTING-ARTIFACT SEARCH
#
# Search ONLY research result/report files.
# Do NOT search raw dataset directories.
# Do NOT load large per-engine prediction tables.
# =============================================================================

fd002_results_root_631d = Path(
    r"C:\RUL_Reliability_Research\06_results"
)

if not fd002_results_root_631d.is_dir():
    raise RuntimeError(
        "06_results directory not found."
    )


fd002_candidate_files_631d = []


def _fd002_reporting_filename_631d(
    path,
):
    lower = path.name.lower()

    if "fd002" not in lower:
        return False

    return any(
        token in lower
        for token in (
            "final",
            "evidence",
            "closure",
            "conclusion",
            "score_summary",
            "scoring_metadata",
            "summary",
            "metadata",
            "report",
        )
    )


for path_631d in fd002_results_root_631d.rglob("*"):

    if not path_631d.is_file():
        continue

    if path_631d.suffix.lower() not in (
        ".json",
        ".csv",
    ):
        continue

    if not _fd002_reporting_filename_631d(
        path_631d
    ):
        continue

    try:
        file_size_631d = path_631d.stat().st_size
    except OSError:
        continue

    # Reporting summaries should be small. Avoid loading large engine-level
    # artifacts in this authority-discrimination step.
    if file_size_631d > 5_000_000:
        continue

    fd002_candidate_files_631d.append(
        path_631d
    )


# =============================================================================
# 5) LOAD FD002 JSON / SUMMARY CSV CANDIDATES
# =============================================================================

fd002_artifacts_631d = []


def _file_sha256_631d(
    path,
):
    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


def _csv_summary_payload_631d(
    path,
):
    # Read only compact summary CSVs.
    # No arithmetic is performed.
    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as handle:
        reader = csv.DictReader(
            handle
        )

        rows = []

        for idx, row in enumerate(
            reader
        ):
            rows.append(
                dict(row)
            )

            # Authority discrimination only.
            # If a file contains hundreds/thousands of rows it is not
            # a compact reporting summary and should not be retained here.
            if idx >= 49:
                break

    return {
        "fieldnames":
            tuple(
                reader.fieldnames
                or
                ()
            ),

        "rows":
            tuple(
                rows
            ),
    }


for path_631d in sorted(
    fd002_candidate_files_631d
):

    suffix_631d = path_631d.suffix.lower()

    try:

        if suffix_631d == ".json":

            with open(
                path_631d,
                "r",
                encoding="utf-8",
            ) as handle:
                payload_631d = json.load(
                    handle_631d
                )

            row_count_631d = None

        else:

            payload_631d = _csv_summary_payload_631d(
                path_631d
            )

            row_count_631d = len(
                payload_631d[
                    "rows"
                ]
            )

    except Exception as exc_631d:

        fd002_artifacts_631d.append(
            {
                "path":
                    str(
                        path_631d
                    ),

                "load_error":
                    repr(
                        exc_631d
                    ),
            }
        )

        continue


    (
        target_convention_631d,
        target_evidence_631d,
    ) = _target_signature_631d(
        payload_631d
    )


    reporting_fields_631d = (
        _extract_reporting_scalars_631d(
            payload_631d
        )
    )


    text_for_score_631d = json.dumps(
        payload_631d,
        default=str,
    ).lower()


    score_631d = 0

    for token, weight in (
        ("official_test", 40),
        ("heldout", 40),
        ("score", 25),
        ("scoring", 25),
        ("final", 30),
        ("full_test_mae", 50),
        ("accepted_mae", 50),
        ("abstained_mae", 50),
        ("selective", 30),
        ("coverage", 20),
        ("frozen_risk_threshold", 20),
        ("accepted_count", 20),
        ("abstained_count", 20),
        ("259", 5),
    ):
        if token in text_for_score_631d:
            score_631d += weight


    fd002_artifacts_631d.append(
        {
            "path":
                str(
                    path_631d
                ),

            "filename":
                path_631d.name,

            "suffix":
                suffix_631d,

            "file_sha256":
                _file_sha256_631d(
                    path_631d
                ),

            "file_size":
                int(
                    path_631d.stat().st_size
                ),

            "loaded_row_count_if_csv":
                row_count_631d,

            "score":
                int(
                    score_631d
                ),

            "target_convention":
                target_convention_631d,

            "target_evidence":
                tuple(
                    target_evidence_631d[
                        :30
                    ]
                ),

            "reporting_fields":
                tuple(
                    reporting_fields_631d[
                        :80
                    ]
                ),
        }
    )


fd002_artifacts_631d = sorted(
    fd002_artifacts_631d,
    key=lambda row: (
        -row.get(
            "score",
            -1,
        ),
        row.get(
            "path",
            "",
        ),
    ),
)


# =============================================================================
# 6) SPECIAL CONSISTENCY SIGNATURE:
# FROZEN FD002 POLICY DECISION COUNTS / THRESHOLD
#
# These are read from the already-recovered decision metadata FILE,
# not typed-in scientific outcomes.
# =============================================================================

fd002_decision_metadata_path_631d = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\step6_reliability_signals"
    r"\fd002_c5\fd002_c5_test_operating_decisions_metadata.json"
)

fd002_decision_signature_631d = None


if fd002_decision_metadata_path_631d.is_file():

    with open(
        fd002_decision_metadata_path_631d,
        "r",
        encoding="utf-8",
    ) as handle_631d:

        fd002_decision_payload_631d = json.load(
            handle_631d
        )


    fd002_decision_signature_631d = {
        "accepted_count":
            fd002_decision_payload_631d.get(
                "accepted_count"
            ),

        "abstained_count":
            fd002_decision_payload_631d.get(
                "abstained_count"
            ),

        "nominal_coverage":
            fd002_decision_payload_631d.get(
                "nominal_coverage"
            ),

        "frozen_risk_threshold":
            fd002_decision_payload_631d.get(
                "frozen_risk_threshold"
            ),

        "official_test_labels_read":
            fd002_decision_payload_631d.get(
                "official_test_labels_read"
            ),

        "decision_rule":
            fd002_decision_payload_631d.get(
                "decision_rule"
            ),

        "file_sha256":
            _file_sha256_631d(
                fd002_decision_metadata_path_631d
            ),
    }


# =============================================================================
# 7) FREEZE AUTHORITY-DISCRIMINATION AUDIT
# =============================================================================

FD001_FINAL_REPORTING_AUTHORITY_631D = {
    "path":
        str(
            fd001_final_path_631d
        ),

    "file_sha256":
        fd001_final_file_sha_631d,

    "target_convention":
        fd001_target_convention_631d,

    "target_evidence":
        fd001_target_evidence_631d,

    "reporting_fields":
        fd001_reporting_fields_631d,
}


FD002_FINAL_REPORTING_AUTHORITY_CANDIDATES_631D = tuple(
    fd002_artifacts_631d
)


FD002_FROZEN_TEST_DECISION_SIGNATURE_631D = (
    fd002_decision_signature_631d
)


audit_payload_631d = {
    "stage":
        "6.31D",

    "status":
        "FD001_FD002_FINAL_AUTHORITY_TARGET_CONVENTION_DISCRIMINATION_COMPLETE",

    "fd001_authority_path":
        str(
            fd001_final_path_631d
        ),

    "fd001_authority_file_sha256":
        fd001_final_file_sha_631d,

    "fd001_target_convention":
        fd001_target_convention_631d,

    "fd002_candidate_count":
        len(
            fd002_artifacts_631d
        ),

    "fd002_decision_signature_available":
        bool(
            fd002_decision_signature_631d
            is not None
        ),

    "cross_dataset_table_constructed":
        False,

    "raw_test_rul_access":
        False,

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_change":
        False,

    "reporting_only":
        True,
}


FD004_FINAL_AUTHORITY_TARGET_AUDIT_631D = (
    audit_payload_631d
)

FD004_FINAL_AUTHORITY_TARGET_AUDIT_SHA256_631D = (
    _json_sha256_631d(
        audit_payload_631d
    )
)

all_checks_passed_631d = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31D — FD001 / FD002 FINAL AUTHORITY + "
    "TARGET-CONVENTION DISCRIMINATOR"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Raw TEST/RUL access               :", False)
print("Threshold/policy change           :", False)
print("Cross-dataset table built         :", False)
print("Reporting-only                    :", True)


print("\n--- FD001 FINAL AUTHORITY ---")

print(
    "Path                              :",
    fd001_final_path_631d,
)

print(
    "File SHA-256                      :",
    fd001_final_file_sha_631d,
)

print(
    "Target convention                 :",
    fd001_target_convention_631d,
)

print(
    "Target evidence:"
)

for path_631d, value_631d in fd001_target_evidence_631d:
    print(
        f"  {path_631d} = {value_631d}"
    )


print("\nFD001 reporting fields:")

for path_631d, value_631d in fd001_reporting_fields_631d:
    print(
        f"  {path_631d} = {value_631d}"
    )


print("\n--- FD002 FROZEN TEST DECISION SIGNATURE ---")

print(
    fd002_decision_signature_631d
)


print("\n--- FD002 FINAL / SCORING AUTHORITY CANDIDATES ---")

print(
    "Candidate count                   :",
    len(
        fd002_artifacts_631d
    ),
)


for idx_631d, row_631d in enumerate(
    fd002_artifacts_631d[
        :20
    ],
    start=1,
):

    print(
        f"\nCANDIDATE {idx_631d}"
    )

    print(
        "  path                            :",
        row_631d.get(
            "path"
        ),
    )

    if "load_error" in row_631d:
        print(
            "  load error                      :",
            row_631d[
                "load_error"
            ],
        )
        continue

    print(
        "  score                           :",
        row_631d[
            "score"
        ],
    )

    print(
        "  file SHA-256                    :",
        row_631d[
            "file_sha256"
        ],
    )

    print(
        "  target convention               :",
        row_631d[
            "target_convention"
        ],
    )

    if row_631d[
        "loaded_row_count_if_csv"
    ] is not None:
        print(
            "  loaded CSV rows                 :",
            row_631d[
                "loaded_row_count_if_csv"
            ],
        )

    print(
        "  target evidence:"
    )

    for path_631d, value_631d in row_631d[
        "target_evidence"
    ]:
        print(
            f"    {path_631d} = {value_631d}"
        )

    print(
        "  reporting fields:"
    )

    for path_631d, value_631d in row_631d[
        "reporting_fields"
    ]:
        print(
            f"    {path_631d} = {value_631d}"
        )


print("\n--- DISCRIMINATION AUDIT ---")

print(
    "FD001 target convention           :",
    fd001_target_convention_631d,
)

print(
    "FD002 candidate count             :",
    len(
        fd002_artifacts_631d
    ),
)

print(
    "Cross-dataset metrics combined    :",
    False,
)

print(
    "Audit SHA-256                     :",
    FD004_FINAL_AUTHORITY_TARGET_AUDIT_SHA256_631D,
)


print(
    "\nPASS — FD001/FD002 reporting authorities were inspected "
    "with target convention kept explicit."
)

print(
    "No cross-dataset numerical comparison has yet been made."
)

print(
    "Next: select the FD002 final authority that matches the frozen "
    "decision signature and establish which FD001–FD004 metrics are "
    "scientifically comparable versus target-convention-specific."
)

In [ ]:
# STEP 6.31D-R1 — FD002 JSON LOADER REPAIR + FINAL OUTCOME AUTHORITY AUDIT
#
# Repair-only / reporting-only step.
#
# The 6.31D JSON failures were caused by:
#     json.load(handle_631d)
# referencing an already-closed handle instead of the newly opened file handle.
#
# This cell:
# - verifies the 6.31D audit
# - reloads FD002 JSON reporting artifacts correctly
# - inspects compact CSV summaries
# - searches small FD002 result artifacts for frozen TEST outcome evidence
# - compares candidate metadata to the already-frozen TEST decision signature
# - keeps raw-vs-capped target convention explicit
#
# NO model inference.
# NO MC dropout.
# NO corruption execution.
# NO raw TEST/RUL access.
# NO metric recomputation.
# NO threshold/policy change.
# NO cross-dataset numerical synthesis yet.

import os
import re
import csv
import json
import hashlib
import numbers
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631dr1 = (
    "all_checks_passed_631d",

    "FD004_FINAL_AUTHORITY_TARGET_AUDIT_631D",
    "FD004_FINAL_AUTHORITY_TARGET_AUDIT_SHA256_631D",

    "FD002_FROZEN_TEST_DECISION_SIGNATURE_631D",
)

missing_631dr1 = [
    name
    for name in required_631dr1
    if name not in globals()
]

if missing_631dr1:
    raise RuntimeError(
        "Missing STEP 6.31D-R1 authority object(s): "
        + ", ".join(missing_631dr1)
    )

if not bool(all_checks_passed_631d):
    raise RuntimeError(
        "STEP 6.31D must remain passed."
    )


def _json_sha256_631dr1(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631dr1(
        FD004_FINAL_AUTHORITY_TARGET_AUDIT_631D
    )
    !=
    str(
        FD004_FINAL_AUTHORITY_TARGET_AUDIT_SHA256_631D
    )
):
    raise RuntimeError(
        "STEP 6.31D authority/target audit changed."
    )


decision_signature_631dr1 = dict(
    FD002_FROZEN_TEST_DECISION_SIGNATURE_631D
)

required_decision_keys_631dr1 = (
    "accepted_count",
    "abstained_count",
    "nominal_coverage",
    "frozen_risk_threshold",
    "official_test_labels_read",
    "decision_rule",
)

missing_decision_keys_631dr1 = [
    key
    for key in required_decision_keys_631dr1
    if key not in decision_signature_631dr1
]

if missing_decision_keys_631dr1:
    raise RuntimeError(
        "FD002 frozen decision signature missing key(s): "
        + ", ".join(missing_decision_keys_631dr1)
    )


frozen_accepted_count_631dr1 = int(
    decision_signature_631dr1[
        "accepted_count"
    ]
)

frozen_abstained_count_631dr1 = int(
    decision_signature_631dr1[
        "abstained_count"
    ]
)

frozen_endpoint_count_631dr1 = (
    frozen_accepted_count_631dr1
    +
    frozen_abstained_count_631dr1
)

frozen_threshold_631dr1 = float(
    decision_signature_631dr1[
        "frozen_risk_threshold"
    ]
)

frozen_nominal_coverage_631dr1 = float(
    decision_signature_631dr1[
        "nominal_coverage"
    ]
)


# =============================================================================
# 1) HELPERS
# =============================================================================

def _file_sha256_631dr1(path):
    with open(
        path,
        "rb",
    ) as handle_file_631dr1:
        return hashlib.sha256(
            handle_file_631dr1.read()
        ).hexdigest()


def _flatten_scalars_631dr1(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 10:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _flatten_scalars_631dr1(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) <= 300:
            for idx, child in enumerate(
                value
            ):
                _flatten_scalars_631dr1(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        output.append(
            (
                ".".join(path),
                value,
            )
        )

    return output


def _target_signature_631dr1(
    payload,
):
    scalars = _flatten_scalars_631dr1(
        payload
    )

    text = "\n".join(
        f"{path}={value}"
        for path, value
        in scalars
    ).lower()

    raw_signals = (
        "rul_raw",
        "raw_rul",
        "official_rul_raw",
        "raw rul",
        "uncapped",
    )

    capped_signals = (
        "rul_cap_125",
        "cap_125",
        "capped_125",
        "piecewise-capped",
        "piecewise capped",
        "capped rul",
    )

    has_raw = any(
        signal in text
        for signal in raw_signals
    )

    has_capped = any(
        signal in text
        for signal in capped_signals
    )

    if has_raw and has_capped:
        convention = "BOTH_RAW_AND_CAPPED_REFERENCED"

    elif has_raw:
        convention = "RAW_RUL_REFERENCED"

    elif has_capped:
        convention = "CAPPED_RUL_REFERENCED"

    else:
        convention = "TARGET_CONVENTION_NOT_EXPLICIT"

    evidence = tuple(
        (
            path,
            value,
        )
        for path, value
        in scalars
        if any(
            token
            in (
                path
                +
                " "
                +
                str(value)
            ).lower()
            for token
            in (
                "target",
                "rul_raw",
                "raw_rul",
                "official_rul",
                "cap_125",
                "capped",
                "piecewise",
            )
        )
    )

    return (
        convention,
        evidence,
    )


metric_path_re_631dr1 = re.compile(
    r"endpoint|"
    r"engine_count|"
    r"test_engines|"
    r"accepted_count|"
    r"abstained_count|"
    r"coverage|"
    r"threshold|"
    r"full.*mae|"
    r"accepted.*mae|"
    r"abstained.*mae|"
    r"selective.*gain|"
    r"relative.*reduction|"
    r"full.*rmse|"
    r"accepted.*rmse|"
    r"abstained.*rmse|"
    r"nasa.*score|"
    r"status|"
    r"evaluation_target|"
    r"target_convention",
    flags=re.IGNORECASE,
)


def _reporting_fields_631dr1(
    payload,
):
    return tuple(
        (
            path,
            value,
        )
        for path, value
        in _flatten_scalars_631dr1(
            payload
        )
        if metric_path_re_631dr1.search(
            path
        )
    )


def _find_scalar_values_631dr1(
    payload,
    key_pattern,
):
    regex = re.compile(
        key_pattern,
        flags=re.IGNORECASE,
    )

    return tuple(
        (
            path,
            value,
        )
        for path, value
        in _flatten_scalars_631dr1(
            payload
        )
        if regex.search(
            path
        )
    )


# =============================================================================
# 2) SEARCH FD002 RESULT TREE FOR SMALL REPORTING ARTIFACTS
# =============================================================================

fd002_result_roots_631dr1 = (
    Path(
        r"C:\RUL_Reliability_Research"
        r"\06_results\step6_reliability_signals"
    ),
    Path(
        r"C:\RUL_Reliability_Research"
        r"\06_results"
    ),
)


candidate_paths_631dr1 = set()


for root_631dr1 in fd002_result_roots_631dr1:

    if not root_631dr1.is_dir():
        continue

    for path_631dr1 in root_631dr1.rglob("*"):

        if not path_631dr1.is_file():
            continue

        suffix_631dr1 = (
            path_631dr1.suffix.lower()
        )

        if suffix_631dr1 not in (
            ".json",
            ".csv",
        ):
            continue

        lower_path_631dr1 = str(
            path_631dr1
        ).lower()

        if "fd002" not in lower_path_631dr1:
            continue

        try:
            size_631dr1 = path_631dr1.stat().st_size
        except OSError:
            continue

        # Reporting artifacts only.
        if size_631dr1 > 5_000_000:
            continue

        filename_lower_631dr1 = (
            path_631dr1.name.lower()
        )

        if not any(
            token in filename_lower_631dr1
            for token in (
                "final",
                "evidence",
                "closure",
                "conclusion",
                "score",
                "scoring",
                "summary",
                "metadata",
                "decision",
                "policy",
            )
        ):
            continue

        candidate_paths_631dr1.add(
            path_631dr1
        )


candidate_paths_631dr1 = sorted(
    candidate_paths_631dr1
)


# =============================================================================
# 3) LOAD CANDIDATES CORRECTLY
# =============================================================================

loaded_artifacts_631dr1 = []


for path_631dr1 in candidate_paths_631dr1:

    suffix_631dr1 = (
        path_631dr1.suffix.lower()
    )

    try:

        if suffix_631dr1 == ".json":

            # BUG FIX:
            # use the handle opened in THIS block.
            with open(
                path_631dr1,
                "r",
                encoding="utf-8",
            ) as handle_json_631dr1:

                payload_631dr1 = json.load(
                    handle_json_631dr1
                )

            compact_csv_rows_631dr1 = None

        else:

            with open(
                path_631dr1,
                "r",
                encoding="utf-8-sig",
                errors="replace",
                newline="",
            ) as handle_csv_631dr1:

                reader_631dr1 = csv.DictReader(
                    handle_csv_631dr1
                )

                rows_631dr1 = []

                for idx_631dr1, row_631dr1 in enumerate(
                    reader_631dr1
                ):

                    rows_631dr1.append(
                        dict(
                            row_631dr1
                        )
                    )

                    # Compact reporting summary only.
                    if idx_631dr1 >= 49:
                        break

            payload_631dr1 = {
                "fieldnames":
                    tuple(
                        reader_631dr1.fieldnames
                        or
                        ()
                    ),

                "rows":
                    tuple(
                        rows_631dr1
                    ),
            }

            compact_csv_rows_631dr1 = tuple(
                rows_631dr1
            )

    except Exception as exc_631dr1:

        loaded_artifacts_631dr1.append(
            {
                "path":
                    str(
                        path_631dr1
                    ),

                "filename":
                    path_631dr1.name,

                "load_error":
                    repr(
                        exc_631dr1
                    ),
            }
        )

        continue


    (
        target_convention_631dr1,
        target_evidence_631dr1,
    ) = _target_signature_631dr1(
        payload_631dr1
    )


    reporting_fields_631dr1 = (
        _reporting_fields_631dr1(
            payload_631dr1
        )
    )


    accepted_values_631dr1 = (
        _find_scalar_values_631dr1(
            payload_631dr1,
            r"accepted_count$",
        )
    )

    abstained_values_631dr1 = (
        _find_scalar_values_631dr1(
            payload_631dr1,
            r"abstained_count$",
        )
    )

    threshold_values_631dr1 = (
        _find_scalar_values_631dr1(
            payload_631dr1,
            r"(?:frozen.*threshold|risk_threshold|threshold)$",
        )
    )

    endpoint_values_631dr1 = (
        _find_scalar_values_631dr1(
            payload_631dr1,
            r"(?:endpoint_count|test_engines|engine_count)$",
        )
    )


    accepted_match_631dr1 = any(
        isinstance(
            value,
            numbers.Number,
        )
        and
        int(value)
        ==
        frozen_accepted_count_631dr1
        for _, value
        in accepted_values_631dr1
    )

    abstained_match_631dr1 = any(
        isinstance(
            value,
            numbers.Number,
        )
        and
        int(value)
        ==
        frozen_abstained_count_631dr1
        for _, value
        in abstained_values_631dr1
    )

    threshold_match_631dr1 = any(
        isinstance(
            value,
            numbers.Number,
        )
        and
        abs(
            float(value)
            -
            frozen_threshold_631dr1
        )
        <=
        1e-15
        for _, value
        in threshold_values_631dr1
    )

    endpoint_match_631dr1 = any(
        isinstance(
            value,
            numbers.Number,
        )
        and
        int(value)
        ==
        frozen_endpoint_count_631dr1
        for _, value
        in endpoint_values_631dr1
    )


    lower_payload_text_631dr1 = json.dumps(
        payload_631dr1,
        default=str,
    ).lower()


    score_631dr1 = 0

    if accepted_match_631dr1:
        score_631dr1 += 60

    if abstained_match_631dr1:
        score_631dr1 += 60

    if threshold_match_631dr1:
        score_631dr1 += 60

    if endpoint_match_631dr1:
        score_631dr1 += 40

    for token, weight in (
        ("official_test", 30),
        ("official test", 30),
        ("full_test_mae", 50),
        ("full_mae", 40),
        ("accepted_mae", 50),
        ("abstained_mae", 50),
        ("selective_mae_gain", 50),
        ("relative_mae_reduction", 40),
        ("evaluation_target", 30),
        ("piecewise-capped", 30),
        ("rul_raw", 30),
        ("raw_rul", 30),
        ("reporting_only", 20),
        ("final", 20),
    ):
        if token in lower_payload_text_631dr1:
            score_631dr1 += weight


    loaded_artifacts_631dr1.append(
        {
            "path":
                str(
                    path_631dr1
                ),

            "filename":
                path_631dr1.name,

            "file_sha256":
                _file_sha256_631dr1(
                    path_631dr1
                ),

            "file_size":
                int(
                    path_631dr1.stat().st_size
                ),

            "suffix":
                suffix_631dr1,

            "score":
                int(
                    score_631dr1
                ),

            "target_convention":
                target_convention_631dr1,

            "target_evidence":
                tuple(
                    target_evidence_631dr1[
                        :40
                    ]
                ),

            "reporting_fields":
                tuple(
                    reporting_fields_631dr1[
                        :100
                    ]
                ),

            "accepted_count_match":
                bool(
                    accepted_match_631dr1
                ),

            "abstained_count_match":
                bool(
                    abstained_match_631dr1
                ),

            "threshold_match":
                bool(
                    threshold_match_631dr1
                ),

            "endpoint_count_match":
                bool(
                    endpoint_match_631dr1
                ),

            "compact_csv_rows":
                compact_csv_rows_631dr1,
        }
    )


loaded_artifacts_631dr1 = sorted(
    loaded_artifacts_631dr1,
    key=lambda row: (
        -row.get(
            "score",
            -1,
        ),
        row.get(
            "path",
            "",
        ),
    ),
)


# =============================================================================
# 4) IDENTIFY EXACT DECISION-SIGNATURE MATCHES
# =============================================================================

exact_signature_matches_631dr1 = tuple(
    row
    for row
    in loaded_artifacts_631dr1
    if "load_error" not in row
    and
    row[
        "accepted_count_match"
    ]
    and
    row[
        "abstained_count_match"
    ]
    and
    row[
        "threshold_match"
    ]
)


partial_signature_matches_631dr1 = tuple(
    row
    for row
    in loaded_artifacts_631dr1
    if "load_error" not in row
    and
    (
        row[
            "accepted_count_match"
        ]
        or
        row[
            "abstained_count_match"
        ]
        or
        row[
            "threshold_match"
        ]
        or
        row[
            "endpoint_count_match"
        ]
    )
)


# =============================================================================
# 5) SPECIAL INSPECTION OF KNOWN OFFICIAL SCORING METADATA
# =============================================================================

known_scoring_metadata_path_631dr1 = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\step6_reliability_signals"
    r"\fd002_c5"
    r"\fd002_c5_official_test_scoring_metadata.json"
)


known_scoring_metadata_631dr1 = None


if known_scoring_metadata_path_631dr1.is_file():

    with open(
        known_scoring_metadata_path_631dr1,
        "r",
        encoding="utf-8",
    ) as handle_known_scoring_631dr1:

        known_scoring_metadata_631dr1 = json.load(
            handle_known_scoring_631dr1
        )


# =============================================================================
# 6) SPECIAL INSPECTION OF 3-ROW SCORE SUMMARY
# =============================================================================

known_score_summary_path_631dr1 = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\step6_reliability_signals"
    r"\fd002_c5"
    r"\fd002_c5_official_test_score_summary.csv"
)


known_score_summary_rows_631dr1 = None


if known_score_summary_path_631dr1.is_file():

    with open(
        known_score_summary_path_631dr1,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as handle_summary_631dr1:

        reader_summary_631dr1 = csv.DictReader(
            handle_summary_631dr1
        )

        known_score_summary_rows_631dr1 = tuple(
            dict(row)
            for row
            in reader_summary_631dr1
        )


# =============================================================================
# 7) FREEZE REPAIR AUDIT
# =============================================================================

FD002_FINAL_OUTCOME_AUTHORITY_CANDIDATES_631DR1 = tuple(
    loaded_artifacts_631dr1
)

FD002_EXACT_DECISION_SIGNATURE_MATCHES_631DR1 = (
    exact_signature_matches_631dr1
)

FD002_PARTIAL_DECISION_SIGNATURE_MATCHES_631DR1 = (
    partial_signature_matches_631dr1
)

FD002_KNOWN_SCORING_METADATA_631DR1 = (
    known_scoring_metadata_631dr1
)

FD002_KNOWN_SCORE_SUMMARY_ROWS_631DR1 = (
    known_score_summary_rows_631dr1
)


audit_payload_631dr1 = {
    "stage":
        "6.31D-R1",

    "status":
        "FD002_JSON_LOADER_REPAIRED_FINAL_OUTCOME_AUTHORITY_AUDITED",

    "frozen_endpoint_count":
        frozen_endpoint_count_631dr1,

    "frozen_accepted_count":
        frozen_accepted_count_631dr1,

    "frozen_abstained_count":
        frozen_abstained_count_631dr1,

    "frozen_nominal_coverage":
        frozen_nominal_coverage_631dr1,

    "frozen_risk_threshold":
        frozen_threshold_631dr1,

    "candidate_count":
        len(
            loaded_artifacts_631dr1
        ),

    "load_error_count":
        sum(
            1
            for row
            in loaded_artifacts_631dr1
            if "load_error" in row
        ),

    "exact_signature_match_count":
        len(
            exact_signature_matches_631dr1
        ),

    "partial_signature_match_count":
        len(
            partial_signature_matches_631dr1
        ),

    "raw_test_rul_access":
        False,

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "corruption_execution":
        False,

    "threshold_policy_change":
        False,

    "cross_dataset_table_constructed":
        False,

    "reporting_only":
        True,
}


FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_631DR1 = (
    audit_payload_631dr1
)

FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_SHA256_631DR1 = (
    _json_sha256_631dr1(
        audit_payload_631dr1
    )
)

all_checks_passed_631dr1 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31D-R1 — FD002 JSON LOADER REPAIR + "
    "FINAL OUTCOME AUTHORITY AUDIT"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Corruption execution              :", False)
print("Raw TEST/RUL access               :", False)
print("Threshold/policy change           :", False)
print("Cross-dataset table built         :", False)
print("Reporting-only                    :", True)


print("\n--- FROZEN FD002 DECISION SIGNATURE ---")

print(
    "Endpoint count                    :",
    frozen_endpoint_count_631dr1,
)

print(
    "Accepted / abstained              :",
    (
        frozen_accepted_count_631dr1,
        frozen_abstained_count_631dr1,
    ),
)

print(
    "Nominal coverage                  :",
    frozen_nominal_coverage_631dr1,
)

print(
    "Frozen threshold                  :",
    frozen_threshold_631dr1,
)

print(
    "Decision rule                     :",
    decision_signature_631dr1[
        "decision_rule"
    ],
)


print("\n--- LOADER REPAIR ---")

print(
    "Candidate artifacts loaded        :",
    len(
        loaded_artifacts_631dr1
    ),
)

print(
    "Load errors after repair          :",
    sum(
        1
        for row
        in loaded_artifacts_631dr1
        if "load_error" in row
    ),
)

print(
    "Exact decision-signature matches  :",
    len(
        exact_signature_matches_631dr1
    ),
)

print(
    "Partial signature matches         :",
    len(
        partial_signature_matches_631dr1
    ),
)


print("\n--- HIGHEST-VALUE FD002 AUTHORITIES ---")

for idx_631dr1, row_631dr1 in enumerate(
    loaded_artifacts_631dr1[
        :20
    ],
    start=1,
):

    print(
        f"\nCANDIDATE {idx_631dr1}"
    )

    print(
        "  path                            :",
        row_631dr1.get(
            "path"
        ),
    )

    if "load_error" in row_631dr1:

        print(
            "  LOAD ERROR                      :",
            row_631dr1[
                "load_error"
            ],
        )

        continue

    print(
        "  score                           :",
        row_631dr1[
            "score"
        ],
    )

    print(
        "  file SHA-256                    :",
        row_631dr1[
            "file_sha256"
        ],
    )

    print(
        "  target convention               :",
        row_631dr1[
            "target_convention"
        ],
    )

    print(
        "  accepted-count match            :",
        row_631dr1[
            "accepted_count_match"
        ],
    )

    print(
        "  abstained-count match           :",
        row_631dr1[
            "abstained_count_match"
        ],
    )

    print(
        "  threshold match                 :",
        row_631dr1[
            "threshold_match"
        ],
    )

    print(
        "  endpoint-count match            :",
        row_631dr1[
            "endpoint_count_match"
        ],
    )

    if row_631dr1[
        "target_evidence"
    ]:

        print(
            "  target evidence:"
        )

        for path_631dr1, value_631dr1 in row_631dr1[
            "target_evidence"
        ]:
            print(
                f"    {path_631dr1} = {value_631dr1}"
            )

    if row_631dr1[
        "reporting_fields"
    ]:

        print(
            "  reporting fields:"
        )

        for path_631dr1, value_631dr1 in row_631dr1[
            "reporting_fields"
        ]:
            print(
                f"    {path_631dr1} = {value_631dr1}"
            )

    if row_631dr1[
        "compact_csv_rows"
    ] is not None:

        print(
            "  compact CSV rows:"
        )

        for csv_idx_631dr1, csv_row_631dr1 in enumerate(
            row_631dr1[
                "compact_csv_rows"
            ],
            start=1,
        ):
            print(
                f"    ROW {csv_idx_631dr1}: "
                f"{csv_row_631dr1}"
            )


print("\n--- KNOWN OFFICIAL SCORING METADATA ---")

if known_scoring_metadata_631dr1 is None:

    print(
        "Metadata unavailable."
    )

else:

    for (
        path_631dr1,
        value_631dr1,
    ) in _reporting_fields_631dr1(
        known_scoring_metadata_631dr1
    ):

        print(
            f"  {path_631dr1} = {value_631dr1}"
        )

    (
        known_target_convention_631dr1,
        known_target_evidence_631dr1,
    ) = _target_signature_631dr1(
        known_scoring_metadata_631dr1
    )

    print(
        "  TARGET CONVENTION               :",
        known_target_convention_631dr1,
    )

    print(
        "  TARGET EVIDENCE:"
    )

    for (
        path_631dr1,
        value_631dr1,
    ) in known_target_evidence_631dr1:

        print(
            f"    {path_631dr1} = {value_631dr1}"
        )


print("\n--- KNOWN 3-ROW SCORE SUMMARY ---")

if known_score_summary_rows_631dr1 is None:

    print(
        "Score summary unavailable."
    )

else:

    print(
        "Row count                         :",
        len(
            known_score_summary_rows_631dr1
        ),
    )

    for idx_631dr1, row_631dr1 in enumerate(
        known_score_summary_rows_631dr1,
        start=1,
    ):

        print(
            f"ROW {idx_631dr1}:",
            row_631dr1,
        )


print("\n--- REPAIR AUDIT ---")

print(
    "Load errors after repair          :",
    audit_payload_631dr1[
        "load_error_count"
    ],
)

print(
    "Exact signature matches           :",
    audit_payload_631dr1[
        "exact_signature_match_count"
    ],
)

print(
    "Audit SHA-256                     :",
    FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_SHA256_631DR1,
)


print(
    "\nPASS — the 6.31D JSON loader bug is repaired."
)

print(
    "FD002 frozen reporting/scoring artifacts have now been "
    "inspected correctly without recomputing scientific outcomes."
)

print(
    "Next: select the authoritative FD002 outcome record and then "
    "separate cross-dataset RAW-RUL comparisons from capped-RUL-specific "
    "results before constructing any final synthesis table."
)

In [ ]:
# STEP 6.31E — RAW-RUL AUTHORITY RECOVERY + CROSS-DATASET TARGET-COMPATIBILITY AUDIT
#
# Purpose:
# 1) Treat the recovered FD002 C5 official score as CAPPED-RUL evidence only.
# 2) Recover any existing historical FD002 RAW-RUL final scoring/reporting authority.
# 3) Audit target conventions for FD001–FD004 before numerical synthesis.
#
# IMPORTANT:
# We are NOT recomputing RAW-RUL MAE from the 259-row FD002 scoring CSV.
# We only accept RAW-RUL metrics if an existing frozen report/artifact contains them.
#
# REPORTING ONLY.
#
# NO inference.
# NO MC dropout.
# NO corruption execution.
# NO raw C-MAPSS TEST/RUL access.
# NO scoring arithmetic from predictions/labels.
# NO threshold/policy/model changes.

import os
import re
import json
import csv
import hashlib
import numbers
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631e = (
    "all_checks_passed_631dr1",

    "FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_631DR1",
    "FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_SHA256_631DR1",

    "FD001_FINAL_REPORTING_AUTHORITY_631D",

    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",

    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
)

missing_631e = [
    name
    for name in required_631e
    if name not in globals()
]

if missing_631e:
    raise RuntimeError(
        "Missing STEP 6.31E authority object(s): "
        + ", ".join(missing_631e)
    )

if not bool(all_checks_passed_631dr1):
    raise RuntimeError(
        "STEP 6.31D-R1 must remain passed."
    )


def _json_sha256_631e(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631e(
        FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_631DR1
    )
    !=
    str(
        FD002_FINAL_OUTCOME_AUTHORITY_AUDIT_SHA256_631DR1
    )
):
    raise RuntimeError(
        "STEP 6.31D-R1 audit changed."
    )


# =============================================================================
# 1) HELPERS
# =============================================================================

def _file_sha256_631e(path):
    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


def _flatten_scalars_631e(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 12:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _flatten_scalars_631e(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (list, tuple),
    ):
        if len(value) <= 500:
            for idx, child in enumerate(
                value
            ):
                _flatten_scalars_631e(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        output.append(
            (
                ".".join(path),
                value,
            )
        )

    return output


raw_metric_path_re_631e = re.compile(
    r"raw.*(?:full|accepted|abstained).*(?:mae|rmse)|"
    r"(?:full|accepted|abstained).*raw.*(?:mae|rmse)|"
    r"raw_(?:full|accepted|abstained)_(?:mae|rmse)|"
    r"(?:full|accepted|abstained)_raw_(?:mae|rmse)|"
    r"raw.*selective.*gain|"
    r"raw.*relative.*reduction",
    flags=re.IGNORECASE,
)


generic_metric_re_631e = re.compile(
    r"full.*mae|"
    r"accepted.*mae|"
    r"abstained.*mae|"
    r"selective.*gain|"
    r"relative.*reduction|"
    r"coverage|"
    r"accepted.*count|"
    r"abstained.*count|"
    r"endpoint.*count|"
    r"threshold|"
    r"evaluation_target|"
    r"target_convention|"
    r"primary_target|"
    r"secondary_target",
    flags=re.IGNORECASE,
)


def _raw_metric_fields_631e(
    payload,
):
    return tuple(
        (
            path,
            value,
        )
        for path, value
        in _flatten_scalars_631e(
            payload
        )
        if raw_metric_path_re_631e.search(
            path
        )
    )


def _generic_reporting_fields_631e(
    payload,
):
    return tuple(
        (
            path,
            value,
        )
        for path, value
        in _flatten_scalars_631e(
            payload
        )
        if generic_metric_re_631e.search(
            path
        )
    )


def _explicit_target_class_631e(
    payload,
):
    scalars = _flatten_scalars_631e(
        payload
    )

    explicit_target_rows = tuple(
        (
            path,
            value,
        )
        for path, value
        in scalars
        if any(
            token in path.lower()
            for token in (
                "evaluation_target",
                "primary_target",
                "secondary_target",
                "target_convention",
                "target_definition",
                "target_rule",
                "rul_target",
            )
        )
    )

    explicit_text = "\n".join(
        (
            path
            +
            "="
            +
            str(value)
        )
        for path, value
        in explicit_target_rows
    ).lower()

    raw_explicit = any(
        token in explicit_text
        for token in (
            "rul_raw",
            "raw rul",
            "raw_rul",
            "uncapped",
        )
    )

    capped_explicit = any(
        token in explicit_text
        for token in (
            "rul_cap_125",
            "cap_125",
            "piecewise-capped",
            "piecewise capped",
            "capped rul",
        )
    )

    if raw_explicit and capped_explicit:
        target_class = "BOTH_TARGETS_EXPLICIT"

    elif raw_explicit:
        target_class = "RAW_RUL_EXPLICIT"

    elif capped_explicit:
        target_class = "CAPPED_RUL_EXPLICIT"

    else:
        target_class = "TARGET_NOT_EXPLICIT"

    return (
        target_class,
        explicit_target_rows,
    )


# =============================================================================
# 2) BIND FD002 CAPPED AUTHORITY AS CAPPED ONLY
# =============================================================================

fd002_capped_path_631e = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\step6_reliability_signals"
    r"\fd002_c5\fd002_c5_official_test_scoring_metadata.json"
)

if not fd002_capped_path_631e.is_file():
    raise RuntimeError(
        "FD002 official scoring metadata disappeared."
    )


with open(
    fd002_capped_path_631e,
    "r",
    encoding="utf-8",
) as handle:
    fd002_capped_payload_631e = json.load(
        handle
    )


fd002_evaluation_target_631e = str(
    fd002_capped_payload_631e.get(
        "evaluation_target"
    )
)

if "capped" not in fd002_evaluation_target_631e.lower():
    raise RuntimeError(
        "FD002 C5 scoring authority is no longer explicitly capped."
    )


FD002_CAPPED_RUL_FINAL_AUTHORITY_631E = {
    "path":
        str(
            fd002_capped_path_631e
        ),

    "file_sha256":
        _file_sha256_631e(
            fd002_capped_path_631e
        ),

    "evaluation_target":
        fd002_evaluation_target_631e,

    "test_endpoints":
        fd002_capped_payload_631e.get(
            "test_endpoints"
        ),

    "accepted_test_endpoints":
        fd002_capped_payload_631e.get(
            "accepted_test_endpoints"
        ),

    "abstained_test_endpoints":
        fd002_capped_payload_631e.get(
            "abstained_test_endpoints"
        ),

    "realized_test_coverage":
        fd002_capped_payload_631e.get(
            "realized_test_coverage"
        ),

    "full_test_mae":
        fd002_capped_payload_631e.get(
            "full_test_mae"
        ),

    "accepted_test_mae":
        fd002_capped_payload_631e.get(
            "accepted_test_mae"
        ),

    "abstained_test_mae":
        fd002_capped_payload_631e.get(
            "abstained_test_mae"
        ),

    "frozen_exact_risk_threshold":
        fd002_capped_payload_631e.get(
            "frozen_exact_risk_threshold"
        ),

    "threshold_modified_after_test":
        fd002_capped_payload_631e.get(
            "threshold_modified_after_test"
        ),
}


# =============================================================================
# 3) DEEP SEARCH FOR EXISTING FD002 RAW-RUL REPORTING METRICS
#
# Search:
#   - 06_results
#   - 03_notebooks
#
# Exclude 02_dataset entirely.
# No row-level recomputation is performed.
# =============================================================================

search_roots_631e = (
    Path(
        r"C:\RUL_Reliability_Research\06_results"
    ),
    Path(
        r"C:\RUL_Reliability_Research\03_notebooks"
    ),
)


fd002_raw_authority_candidates_631e = []


text_raw_metric_re_631e = re.compile(
    r"raw[_ ]?(?:full|accepted|abstained)[_ ]?(?:mae|rmse)|"
    r"(?:full|accepted|abstained)[_ ]?raw[_ ]?(?:mae|rmse)|"
    r"raw[_ ]?rul.{0,120}(?:mae|rmse)|"
    r"(?:mae|rmse).{0,120}raw[_ ]?rul",
    flags=re.IGNORECASE | re.DOTALL,
)


for root_631e in search_roots_631e:

    if not root_631e.is_dir():
        continue

    for path_631e in root_631e.rglob("*"):

        if not path_631e.is_file():
            continue

        suffix_631e = path_631e.suffix.lower()

        if suffix_631e not in (
            ".json",
            ".csv",
            ".txt",
            ".md",
            ".log",
            ".ipynb",
        ):
            continue

        lower_path_631e = str(
            path_631e
        ).lower()

        # Content or filename must concern FD002.
        filename_has_fd002_631e = (
            "fd002"
            in lower_path_631e
        )

        try:
            size_631e = path_631e.stat().st_size
        except OSError:
            continue

        # Reporting/source artifacts only.
        if size_631e > 25_000_000:
            continue


        # ---------------------------------------------------------------------
        # JSON: preserve structured fields where possible
        # ---------------------------------------------------------------------

        if suffix_631e == ".json":

            try:
                with open(
                    path_631e,
                    "r",
                    encoding="utf-8",
                ) as handle:
                    payload_631e = json.load(
                        handle
                    )
            except Exception:
                continue

            serialized_631e = json.dumps(
                payload_631e,
                default=str,
            )

            if (
                not filename_has_fd002_631e
                and
                "fd002" not in serialized_631e.lower()
            ):
                continue

            raw_fields_631e = (
                _raw_metric_fields_631e(
                    payload_631e
                )
            )

            if not raw_fields_631e:
                continue

            (
                target_class_631e,
                target_rows_631e,
            ) = _explicit_target_class_631e(
                payload_631e
            )

            fd002_raw_authority_candidates_631e.append(
                {
                    "source_type":
                        "JSON",

                    "path":
                        str(
                            path_631e
                        ),

                    "file_sha256":
                        _file_sha256_631e(
                            path_631e
                        ),

                    "target_class":
                        target_class_631e,

                    "target_evidence":
                        target_rows_631e,

                    "raw_metric_fields":
                        raw_fields_631e,

                    "reporting_fields":
                        _generic_reporting_fields_631e(
                            payload_631e
                        ),
                }
            )

            continue


        # ---------------------------------------------------------------------
        # Other text-like files
        # ---------------------------------------------------------------------

        try:
            with open(
                path_631e,
                "r",
                encoding="utf-8",
                errors="ignore",
            ) as handle:
                text_631e = handle.read()
        except Exception:
            continue


        lower_text_631e = text_631e.lower()

        if (
            not filename_has_fd002_631e
            and
            "fd002" not in lower_text_631e
        ):
            continue

        if not text_raw_metric_re_631e.search(
            text_631e
        ):
            continue


        lines_631e = text_631e.splitlines()

        matching_line_numbers_631e = [
            idx
            for idx, line
            in enumerate(
                lines_631e,
                start=1,
            )
            if text_raw_metric_re_631e.search(
                line
            )
        ]


        contexts_631e = []
        occupied_631e = []

        for match_line_631e in matching_line_numbers_631e:

            start_631e = max(
                1,
                match_line_631e - 8,
            )

            end_631e = min(
                len(
                    lines_631e
                ),
                match_line_631e + 14,
            )

            if any(
                not (
                    end_631e < used_start
                    or
                    start_631e > used_end
                )
                for used_start, used_end
                in occupied_631e
            ):
                continue

            occupied_631e.append(
                (
                    start_631e,
                    end_631e,
                )
            )

            contexts_631e.append(
                tuple(
                    (
                        line_no_631e,
                        lines_631e[
                            line_no_631e - 1
                        ],
                    )
                    for line_no_631e
                    in range(
                        start_631e,
                        end_631e + 1,
                    )
                )
            )

            if len(
                contexts_631e
            ) >= 12:
                break


        fd002_raw_authority_candidates_631e.append(
            {
                "source_type":
                    suffix_631e.upper().lstrip(
                        "."
                    ),

                "path":
                    str(
                        path_631e
                    ),

                "file_sha256":
                    _file_sha256_631e(
                        path_631e
                    ),

                "contexts":
                    tuple(
                        contexts_631e
                    ),
            }
        )


# =============================================================================
# 4) SEARCH RESIDENT SOURCE STRINGS FOR HISTORICAL FD002 RAW METRICS
# =============================================================================

resident_raw_sources_631e = []

seen_resident_sha_631e = set()


for name_631e, value_631e in list(
    globals().items()
):

    if not isinstance(
        value_631e,
        str,
    ):
        continue

    if len(value_631e) < 100:
        continue

    lower_631e = value_631e.lower()

    if "fd002" not in lower_631e:
        continue

    if not text_raw_metric_re_631e.search(
        value_631e
    ):
        continue

    sha_631e = hashlib.sha256(
        value_631e.encode(
            "utf-8"
        )
    ).hexdigest()

    if sha_631e in seen_resident_sha_631e:
        continue

    seen_resident_sha_631e.add(
        sha_631e
    )

    lines_631e = value_631e.splitlines()

    matches_631e = [
        idx
        for idx, line
        in enumerate(
            lines_631e,
            start=1,
        )
        if text_raw_metric_re_631e.search(
            line
        )
    ]

    contexts_631e = []

    for match_line_631e in matches_631e[
        :8
    ]:

        start_631e = max(
            1,
            match_line_631e - 8,
        )

        end_631e = min(
            len(
                lines_631e
            ),
            match_line_631e + 14,
        )

        contexts_631e.append(
            tuple(
                (
                    line_no_631e,
                    lines_631e[
                        line_no_631e - 1
                    ],
                )
                for line_no_631e
                in range(
                    start_631e,
                    end_631e + 1,
                )
            )
        )

    resident_raw_sources_631e.append(
        {
            "name":
                name_631e,

            "sha256":
                sha_631e,

            "contexts":
                tuple(
                    contexts_631e
                ),
        }
    )


# =============================================================================
# 5) TARGET-CONVENTION EVIDENCE FOR FD001 / FD003 / FD004
# =============================================================================

# FD001 final authority was already loaded/frozen by 6.31D.
fd001_authority_631e = (
    FD001_FINAL_REPORTING_AUTHORITY_631D
)


# Inspect final resident conclusions/contracts for explicit target terminology.
fd003_combined_text_631e = json.dumps(
    {
        "conclusion":
            FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,

        "closure":
            FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,
    },
    default=str,
).lower()


fd004_combined_text_631e = json.dumps(
    {
        "conclusion":
            FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11,

        "closure":
            FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11,
    },
    default=str,
).lower()


def _coarse_target_terms_631e(
    text,
):
    return {
        "contains_raw_rul_term":
            bool(
                (
                    "raw rul"
                    in text
                )
                or
                (
                    "rul_raw"
                    in text
                )
                or
                (
                    "uncapped"
                    in text
                )
            ),

        "contains_capped_rul_term":
            bool(
                (
                    "capped"
                    in text
                )
                or
                (
                    "cap_125"
                    in text
                )
                or
                (
                    "rul_cap_125"
                    in text
                )
            ),
    }


fd003_target_term_audit_631e = (
    _coarse_target_terms_631e(
        fd003_combined_text_631e
    )
)

fd004_target_term_audit_631e = (
    _coarse_target_terms_631e(
        fd004_combined_text_631e
    )
)


# =============================================================================
# 6) FREEZE AUDIT
# =============================================================================

FD002_RAW_RUL_AUTHORITY_CANDIDATES_631E = tuple(
    fd002_raw_authority_candidates_631e
)

FD002_RAW_RUL_RESIDENT_SOURCE_CANDIDATES_631E = tuple(
    resident_raw_sources_631e
)

FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_631E = {
    "stage":
        "6.31E",

    "status":
        "RAW_RUL_AUTHORITY_RECOVERY_AND_TARGET_COMPATIBILITY_AUDIT_COMPLETE",

    "fd001_final_authority":
        fd001_authority_631e,

    "fd002_capped_authority":
        FD002_CAPPED_RUL_FINAL_AUTHORITY_631E,

    "fd002_raw_authority_candidate_count":
        len(
            fd002_raw_authority_candidates_631e
        ),

    "fd002_resident_raw_source_candidate_count":
        len(
            resident_raw_sources_631e
        ),

    "fd003_target_term_audit":
        fd003_target_term_audit_631e,

    "fd004_target_term_audit":
        fd004_target_term_audit_631e,

    "cross_dataset_numerical_table_constructed":
        False,

    "raw_test_rul_access":
        False,

    "raw_metric_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_change":
        False,

    "reporting_only":
        True,
}


FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_SHA256_631E = (
    _json_sha256_631e(
        FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_631E
    )
)

all_checks_passed_631e = True


# =============================================================================
# 7) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31E — RAW-RUL AUTHORITY RECOVERY + "
    "CROSS-DATASET TARGET-COMPATIBILITY AUDIT"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw metric recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Raw C-MAPSS TEST/RUL access       :", False)
print("Threshold/policy change           :", False)
print("Cross-dataset table built         :", False)
print("Reporting-only                    :", True)


print("\n--- FD002 CAPPED-RUL AUTHORITY ---")

for key_631e, value_631e in (
    FD002_CAPPED_RUL_FINAL_AUTHORITY_631E.items()
):
    print(
        f"{key_631e:35s}:",
        value_631e,
    )


print("\n--- FD002 EXISTING RAW-RUL REPORTING CANDIDATES ---")

print(
    "File/source candidate count       :",
    len(
        fd002_raw_authority_candidates_631e
    ),
)

for idx_631e, row_631e in enumerate(
    fd002_raw_authority_candidates_631e[
        :20
    ],
    start=1,
):

    print(
        f"\nCANDIDATE {idx_631e}"
    )

    print(
        "  source type                     :",
        row_631e[
            "source_type"
        ],
    )

    print(
        "  path                            :",
        row_631e[
            "path"
        ],
    )

    print(
        "  SHA-256                         :",
        row_631e[
            "file_sha256"
        ],
    )

    if "target_class" in row_631e:

        print(
            "  explicit target class           :",
            row_631e[
                "target_class"
            ],
        )

        print(
            "  target evidence:"
        )

        for (
            path_value_631e,
            scalar_value_631e,
        ) in row_631e[
            "target_evidence"
        ]:

            print(
                f"    {path_value_631e} = "
                f"{scalar_value_631e}"
            )

        print(
            "  RAW metric fields:"
        )

        for (
            path_value_631e,
            scalar_value_631e,
        ) in row_631e[
            "raw_metric_fields"
        ]:

            print(
                f"    {path_value_631e} = "
                f"{scalar_value_631e}"
            )

    else:

        for context_631e in row_631e[
            "contexts"
        ][
            :8
        ]:

            print()

            for (
                line_no_631e,
                line_631e,
            ) in context_631e:

                print(
                    f"    "
                    f"{line_no_631e:06d} | "
                    f"{line_631e}"
                )


print("\n--- RESIDENT HISTORICAL FD002 RAW-RUL SOURCE CANDIDATES ---")

print(
    "Candidate count                   :",
    len(
        resident_raw_sources_631e
    ),
)

for idx_631e, row_631e in enumerate(
    resident_raw_sources_631e[
        :10
    ],
    start=1,
):

    print(
        f"\nSOURCE {idx_631e}: "
        f"{row_631e['name']}"
    )

    print(
        "  SHA-256                         :",
        row_631e[
            "sha256"
        ],
    )

    for context_631e in row_631e[
        "contexts"
    ][
        :6
    ]:

        print()

        for (
            line_no_631e,
            line_631e,
        ) in context_631e:

            print(
                f"    "
                f"{line_no_631e:06d} | "
                f"{line_631e}"
            )


print("\n--- CROSS-DATASET TARGET AUDIT ---")

print(
    "FD001 authority target convention :",
    fd001_authority_631e[
        "target_convention"
    ],
)

print(
    "FD003 target-term audit           :",
    fd003_target_term_audit_631e,
)

print(
    "FD004 target-term audit           :",
    fd004_target_term_audit_631e,
)


print("\n--- AUDIT ---")

print(
    "FD002 RAW authority candidates    :",
    len(
        fd002_raw_authority_candidates_631e
    ),
)

print(
    "Cross-dataset metrics combined    :",
    False,
)

print(
    "Audit SHA-256                     :",
    FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_SHA256_631E,
)


print(
    "\nPASS — no cross-dataset numerical comparison was made."
)

print(
    "FD002 C5 official TEST scoring is now explicitly bound as "
    "piecewise-capped-RUL evidence."
)

print(
    "Next: if an existing FD002 RAW-RUL authority was recovered, bind it "
    "without recomputation; then construct separate comparable RAW-RUL and "
    "target-specific synthesis tables."
)

In [ ]:
# STEP 6.31F — TARGET-PROVENANCE CLOSURE + FD002 RAW FALSE-POSITIVE REJECTION
#
# Purpose:
# 1) Formally reject the 6.31E apparent FD002 RAW-RUL candidate as a
#    contaminated false positive from recent notebook output containing
#    FD001 source_H5B.raw_* fields.
# 2) Recover exact resident target provenance for FD003 and FD004.
# 3) Freeze what is and is not numerically comparable across FD001–FD004.
#
# REPORTING / PROVENANCE ONLY.
#
# NO inference.
# NO MC dropout.
# NO corruption execution.
# NO TEST/RUL access.
# NO metric recomputation.
# NO threshold/policy/model change.
# NO final cross-dataset metric table yet.

import re
import json
import hashlib
import numbers


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631f = (
    "all_checks_passed_631e",

    "FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_631E",
    "FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_SHA256_631E",

    "FD002_CAPPED_RUL_FINAL_AUTHORITY_631E",
    "FD002_RAW_RUL_AUTHORITY_CANDIDATES_631E",

    "FD001_FINAL_REPORTING_AUTHORITY_631D",

    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",

    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
)

missing_631f = [
    name
    for name in required_631f
    if name not in globals()
]

if missing_631f:
    raise RuntimeError(
        "Missing STEP 6.31F authority object(s): "
        + ", ".join(missing_631f)
    )

if not bool(all_checks_passed_631e):
    raise RuntimeError(
        "STEP 6.31E must remain passed."
    )


def _json_sha256_631f(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631f(
        FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_631E
    )
    !=
    str(
        FD004_CROSS_DATASET_TARGET_COMPATIBILITY_AUDIT_SHA256_631E
    )
):
    raise RuntimeError(
        "STEP 6.31E target-compatibility audit changed."
    )


# =============================================================================
# 1) FORMALLY REJECT FD002 RAW FALSE POSITIVE
# =============================================================================

fd002_raw_candidates_631f = tuple(
    FD002_RAW_RUL_AUTHORITY_CANDIDATES_631E
)

fd002_raw_candidate_count_631f = len(
    fd002_raw_candidates_631f
)


# The only recovered candidate in 6.31E was the FD002 notebook itself.
# Its displayed "raw_*" values were FD001 source_H5B fields printed by our
# recent 6.31D analysis, not historical FD002 raw scoring metrics.
#
# We do NOT infer from filename alone; inspect the candidate contexts.

fd002_false_positive_evidence_631f = []

for candidate_idx_631f, row_631f in enumerate(
    fd002_raw_candidates_631f,
    start=1,
):

    contexts_631f = tuple(
        row_631f.get(
            "contexts",
            ()
        )
    )

    flattened_context_text_631f = "\n".join(
        line
        for context_631f
        in contexts_631f
        for _, line
        in context_631f
    )

    lower_context_631f = (
        flattened_context_text_631f.lower()
    )

    has_fd001_h5b_raw_fields_631f = bool(
        "source_h5b.raw_full_mae"
        in lower_context_631f
        and
        "source_h5b.raw_accepted_mae"
        in lower_context_631f
        and
        "source_h5b.raw_abstained_mae"
        in lower_context_631f
    )

    transitions_to_fd002_locator_output_631f = bool(
        "fd002 frozen test decision signature"
        in lower_context_631f
        or
        "fd002 final / scoring authority candidates"
        in lower_context_631f
    )

    contains_explicit_fd002_raw_metric_name_631f = bool(
        re.search(
            r"fd002.{0,80}raw[_ ]?(?:full|accepted|abstained)[_ ]?(?:mae|rmse)"
            r"|raw[_ ]?(?:full|accepted|abstained)[_ ]?(?:mae|rmse).{0,80}fd002",
            flattened_context_text_631f,
            flags=re.IGNORECASE | re.DOTALL,
        )
    )


    contaminated_false_positive_631f = bool(
        has_fd001_h5b_raw_fields_631f
        and
        transitions_to_fd002_locator_output_631f
        and
        not contains_explicit_fd002_raw_metric_name_631f
    )


    fd002_false_positive_evidence_631f.append(
        {
            "candidate_index":
                int(
                    candidate_idx_631f
                ),

            "path":
                row_631f.get(
                    "path"
                ),

            "source_type":
                row_631f.get(
                    "source_type"
                ),

            "file_sha256":
                row_631f.get(
                    "file_sha256"
                ),

            "contains_fd001_source_H5B_raw_fields":
                has_fd001_h5b_raw_fields_631f,

            "transitions_to_fd002_recent_locator_output":
                transitions_to_fd002_locator_output_631f,

            "contains_explicit_fd002_raw_metric_name":
                contains_explicit_fd002_raw_metric_name_631f,

            "rejected_as_false_positive":
                contaminated_false_positive_631f,
        }
    )


all_fd002_raw_candidates_rejected_631f = bool(
    fd002_raw_candidate_count_631f
    >
    0
    and
    all(
        row[
            "rejected_as_false_positive"
        ]
        for row
        in fd002_false_positive_evidence_631f
    )
)


if (
    fd002_raw_candidate_count_631f
    >
    0
    and
    not all_fd002_raw_candidates_rejected_631f
):
    raise RuntimeError(
        "At least one FD002 RAW-RUL candidate was not proven to be "
        "the known contaminated false positive. Inspect before proceeding."
    )


FD002_RAW_RUL_FINAL_AUTHORITY_AVAILABLE_631F = False

FD002_RAW_RUL_AUTHORITY_STATUS_631F = (
    "NO_GENUINE_PREEXISTING_FD002_RAW_RUL_FINAL_REPORT_RECOVERED"
)


# =============================================================================
# 2) TARGET-PROVENANCE SEARCH — FD003 / FD004 RESIDENT STRUCTURED OBJECTS
#
# Search contents, not just object names.
# =============================================================================

target_signal_re_631f = re.compile(
    r"target|"
    r"rul|"
    r"uncapped|"
    r"capped|"
    r"cap[_ ]?125|"
    r"piecewise",
    flags=re.IGNORECASE,
)


strong_raw_re_631f = re.compile(
    r"uncapped|"
    r"rul_raw|"
    r"raw[_ ]?rul|"
    r"remaining useful life.*uncapped",
    flags=re.IGNORECASE,
)


strong_capped_re_631f = re.compile(
    r"rul_cap_125|"
    r"cap[_ ]?125|"
    r"piecewise[-_ ]?capped|"
    r"capped[_ ]?rul",
    flags=re.IGNORECASE,
)


def _flatten_target_scalars_631f(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 9:
        return output

    if isinstance(
        value,
        dict,
    ):

        for key, child in value.items():

            _flatten_target_scalars_631f(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        if len(value) <= 300:

            for idx, child in enumerate(
                value
            ):

                _flatten_target_scalars_631f(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):

        path_text = ".".join(
            path
        )

        scalar_text = str(
            value
        )

        if target_signal_re_631f.search(
            path_text
            +
            " "
            +
            scalar_text
        ):
            output.append(
                (
                    path_text,
                    value,
                )
            )

    return output


def _dataset_target_candidates_631f(
    dataset_token,
):
    rows = []

    for name_631f, value_631f in list(
        globals().items()
    ):

        if not isinstance(
            value_631f,
            (
                dict,
                list,
                tuple,
            ),
        ):
            continue

        try:
            serialized_631f = json.dumps(
                value_631f,
                default=str,
            )
        except Exception:
            continue

        if (
            dataset_token.lower()
            not in serialized_631f.lower()
            and
            dataset_token.lower()
            not in name_631f.lower()
        ):
            continue


        target_fields_631f = (
            _flatten_target_scalars_631f(
                value_631f
            )
        )

        if not target_fields_631f:
            continue


        target_text_631f = "\n".join(
            (
                path
                +
                "="
                +
                str(value)
            )
            for path, value
            in target_fields_631f
        )


        raw_signal_631f = bool(
            strong_raw_re_631f.search(
                target_text_631f
            )
        )

        capped_signal_631f = bool(
            strong_capped_re_631f.search(
                target_text_631f
            )
        )


        score_631f = 0

        if raw_signal_631f:
            score_631f += 100

        if capped_signal_631f:
            score_631f += 100

        upper_name_631f = (
            name_631f.upper()
        )

        for token, weight in (
            ("TARGET", 30),
            ("CONTRACT", 20),
            ("TRAIN", 15),
            ("WINDOW", 10),
            ("DATA", 10),
            ("FOUNDATION", 10),
            ("CLOSURE", 5),
        ):
            if token in upper_name_631f:
                score_631f += weight


        rows.append(
            {
                "name":
                    name_631f,

                "score":
                    int(
                        score_631f
                    ),

                "raw_signal":
                    raw_signal_631f,

                "capped_signal":
                    capped_signal_631f,

                "target_fields":
                    tuple(
                        target_fields_631f[
                            :80
                        ]
                    ),
            }
        )


    return sorted(
        rows,
        key=lambda row: (
            -row[
                "score"
            ],
            row[
                "name"
            ],
        ),
    )


fd003_target_candidates_631f = (
    _dataset_target_candidates_631f(
        "FD003"
    )
)

fd004_target_candidates_631f = (
    _dataset_target_candidates_631f(
        "FD004"
    )
)


# =============================================================================
# 3) TARGET-PROVENANCE SEARCH — HISTORICAL SOURCE TEXT
#
# Exclude current 6.31 analysis cells to avoid self-contamination.
# =============================================================================

source_target_hits_631f = {
    "FD003": [],
    "FD004": [],
}


def _recent_synthesis_text_631f(
    text,
):
    lower = text.lower()

    return bool(
        "step 6.31" in lower
        or
        "631a" in lower
        or
        "631b" in lower
        or
        "631c" in lower
        or
        "631d" in lower
        or
        "631e" in lower
        or
        "631f" in lower
    )


source_pool_631f = []


if "In" in globals():

    try:
        for sequence_631f, text_631f in enumerate(
            In
        ):
            if isinstance(
                text_631f,
                str,
            ):
                source_pool_631f.append(
                    (
                        f"In[{sequence_631f}]",
                        text_631f,
                    )
                )
    except Exception:
        pass


for name_631f, value_631f in list(
    globals().items()
):

    if not isinstance(
        value_631f,
        str,
    ):
        continue

    source_pool_631f.append(
        (
            "global:" + name_631f,
            value_631f,
        )
    )


seen_source_sha_631f = set()


for origin_631f, text_631f in source_pool_631f:

    if len(
        text_631f
    ) < 100:
        continue

    if _recent_synthesis_text_631f(
        text_631f
    ):
        continue

    text_lower_631f = text_631f.lower()


    for dataset_631f in (
        "FD003",
        "FD004",
    ):

        if dataset_631f.lower() not in text_lower_631f:
            continue

        raw_signal_631f = bool(
            strong_raw_re_631f.search(
                text_631f
            )
        )

        capped_signal_631f = bool(
            strong_capped_re_631f.search(
                text_631f
            )
        )

        if not (
            raw_signal_631f
            or
            capped_signal_631f
        ):
            continue


        sha_631f = hashlib.sha256(
            text_631f.encode(
                "utf-8"
            )
        ).hexdigest()

        unique_key_631f = (
            dataset_631f,
            sha_631f,
        )

        if unique_key_631f in seen_source_sha_631f:
            continue

        seen_source_sha_631f.add(
            unique_key_631f
        )


        lines_631f = text_631f.splitlines()

        match_lines_631f = [
            idx
            for idx, line
            in enumerate(
                lines_631f,
                start=1,
            )
            if (
                strong_raw_re_631f.search(
                    line
                )
                or
                strong_capped_re_631f.search(
                    line
                )
            )
        ]


        contexts_631f = []
        occupied_631f = []


        for match_line_631f in match_lines_631f:

            start_631f = max(
                1,
                match_line_631f - 7,
            )

            end_631f = min(
                len(
                    lines_631f
                ),
                match_line_631f + 10,
            )

            context_text_631f = "\n".join(
                lines_631f[
                    start_631f - 1:
                    end_631f
                ]
            )

            # Require dataset token reasonably near target terminology.
            if (
                dataset_631f.lower()
                not in context_text_631f.lower()
            ):
                continue

            if any(
                not (
                    end_631f < used_start
                    or
                    start_631f > used_end
                )
                for used_start, used_end
                in occupied_631f
            ):
                continue

            occupied_631f.append(
                (
                    start_631f,
                    end_631f,
                )
            )

            contexts_631f.append(
                tuple(
                    (
                        line_no_631f,
                        lines_631f[
                            line_no_631f - 1
                        ],
                    )
                    for line_no_631f
                    in range(
                        start_631f,
                        end_631f + 1,
                    )
                )
            )

            if len(
                contexts_631f
            ) >= 8:
                break


        source_target_hits_631f[
            dataset_631f
        ].append(
            {
                "origin":
                    origin_631f,

                "sha256":
                    sha_631f,

                "raw_signal":
                    raw_signal_631f,

                "capped_signal":
                    capped_signal_631f,

                "contexts":
                    tuple(
                        contexts_631f
                    ),
            }
        )


# =============================================================================
# 4) RESOLVE TARGET CLASS ONLY FROM EXPLICIT EVIDENCE
# =============================================================================

def _resolve_target_class_631f(
    structured_rows,
    source_rows,
):
    raw_evidence = []
    capped_evidence = []


    for row in structured_rows:

        if row[
            "raw_signal"
        ]:
            raw_evidence.append(
                (
                    "structured",
                    row[
                        "name"
                    ],
                )
            )

        if row[
            "capped_signal"
        ]:
            capped_evidence.append(
                (
                    "structured",
                    row[
                        "name"
                    ],
                )
            )


    for row in source_rows:

        if row[
            "raw_signal"
        ]:
            raw_evidence.append(
                (
                    "source",
                    row[
                        "origin"
                    ],
                )
            )

        if row[
            "capped_signal"
        ]:
            capped_evidence.append(
                (
                    "source",
                    row[
                        "origin"
                    ],
                )
            )


    if raw_evidence and not capped_evidence:

        resolved = (
            "RAW_OR_UNCAPPED_RUL_EXPLICIT"
        )

    elif capped_evidence and not raw_evidence:

        resolved = (
            "CAPPED_RUL_EXPLICIT"
        )

    elif raw_evidence and capped_evidence:

        resolved = (
            "MIXED_TARGET_TERMINOLOGY_REQUIRES_FURTHER_DISCRIMINATION"
        )

    else:

        resolved = (
            "TARGET_NOT_YET_EXPLICITLY_PROVEN"
        )


    return (
        resolved,
        tuple(
            raw_evidence
        ),
        tuple(
            capped_evidence
        ),
    )


(
    fd003_target_class_631f,
    fd003_raw_evidence_631f,
    fd003_capped_evidence_631f,
) = _resolve_target_class_631f(
    fd003_target_candidates_631f,
    source_target_hits_631f[
        "FD003"
    ],
)


(
    fd004_target_class_631f,
    fd004_raw_evidence_631f,
    fd004_capped_evidence_631f,
) = _resolve_target_class_631f(
    fd004_target_candidates_631f,
    source_target_hits_631f[
        "FD004"
    ],
)


# =============================================================================
# 5) CROSS-DATASET TARGET COMPATIBILITY CONTRACT
# =============================================================================

FD004_CROSS_DATASET_TARGET_PROVENANCE_631F = {
    "FD001": {
        "authority":
            FD001_FINAL_REPORTING_AUTHORITY_631D[
                "path"
            ],

        "target_status":
            "BOTH_CAPPED_PRIMARY_AND_RAW_SECONDARY_FINAL_METRICS_AVAILABLE",

        "capped_metrics_available":
            True,

        "raw_metrics_available":
            True,
    },

    "FD002": {
        "capped_authority":
            FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                "path"
            ],

        "capped_target":
            FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                "evaluation_target"
            ],

        "capped_metrics_available":
            True,

        "raw_metrics_available":
            False,

        "raw_authority_status":
            FD002_RAW_RUL_AUTHORITY_STATUS_631F,
    },

    "FD003": {
        "target_class":
            fd003_target_class_631f,

        "raw_evidence":
            fd003_raw_evidence_631f,

        "capped_evidence":
            fd003_capped_evidence_631f,
    },

    "FD004": {
        "target_class":
            fd004_target_class_631f,

        "raw_evidence":
            fd004_raw_evidence_631f,

        "capped_evidence":
            fd004_capped_evidence_631f,
    },

    "comparison_rule":
        (
            "Do not compare absolute MAE values across datasets unless "
            "their target convention is explicitly compatible. "
            "Target-specific results may still be reported side by side "
            "with the target convention shown."
        ),

    "fd002_raw_candidate_rejected":
        True,

    "fd002_raw_candidate_rejection_basis":
        tuple(
            fd002_false_positive_evidence_631f
        ),
}


FD004_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631F = (
    _json_sha256_631f(
        FD004_CROSS_DATASET_TARGET_PROVENANCE_631F
    )
)


audit_payload_631f = {
    "stage":
        "6.31F",

    "status":
        "TARGET_PROVENANCE_CLOSED_FD002_RAW_FALSE_POSITIVE_REJECTED",

    "fd002_raw_candidate_count":
        fd002_raw_candidate_count_631f,

    "fd002_all_raw_candidates_rejected":
        all_fd002_raw_candidates_rejected_631f,

    "fd002_genuine_raw_authority_available":
        False,

    "fd003_target_class":
        fd003_target_class_631f,

    "fd004_target_class":
        fd004_target_class_631f,

    "cross_dataset_absolute_mae_table_constructed":
        False,

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "raw_test_rul_access":
        False,

    "threshold_policy_change":
        False,

    "reporting_only":
        True,
}


FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_631F = (
    audit_payload_631f
)

FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F = (
    _json_sha256_631f(
        audit_payload_631f
    )
)

all_checks_passed_631f = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31F — TARGET-PROVENANCE CLOSURE + "
    "FD002 RAW FALSE-POSITIVE REJECTION"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Model inference                   :", False)
print("MC dropout                        :", False)
print("Raw TEST/RUL access               :", False)
print("Threshold/policy change           :", False)
print("Absolute-MAE synthesis built      :", False)
print("Reporting-only                    :", True)


print("\n--- FD002 RAW-RUL RECOVERY DECISION ---")

print(
    "Apparent RAW candidates           :",
    fd002_raw_candidate_count_631f,
)

for row_631f in fd002_false_positive_evidence_631f:

    print(
        "\nCandidate",
        row_631f[
            "candidate_index"
        ],
    )

    print(
        "  path                            :",
        row_631f[
            "path"
        ],
    )

    print(
        "  contains FD001 H5B raw fields   :",
        row_631f[
            "contains_fd001_source_H5B_raw_fields"
        ],
    )

    print(
        "  transitions to recent FD002 log :",
        row_631f[
            "transitions_to_fd002_recent_locator_output"
        ],
    )

    print(
        "  explicit FD002 raw metric name  :",
        row_631f[
            "contains_explicit_fd002_raw_metric_name"
        ],
    )

    print(
        "  rejected as false positive      :",
        row_631f[
            "rejected_as_false_positive"
        ],
    )


print(
    "\nGenuine FD002 RAW authority       :",
    FD002_RAW_RUL_FINAL_AUTHORITY_AVAILABLE_631F,
)

print(
    "FD002 RAW authority status        :",
    FD002_RAW_RUL_AUTHORITY_STATUS_631F,
)


print("\n--- FD003 TARGET-PROVENANCE CANDIDATES ---")

print(
    "Structured candidates             :",
    len(
        fd003_target_candidates_631f
    ),
)

for idx_631f, row_631f in enumerate(
    fd003_target_candidates_631f[
        :10
    ],
    start=1,
):

    print(
        f"\nFD003 STRUCTURED {idx_631f}: "
        f"{row_631f['name']}"
    )

    print(
        "  score                           :",
        row_631f[
            "score"
        ],
    )

    print(
        "  raw signal                      :",
        row_631f[
            "raw_signal"
        ],
    )

    print(
        "  capped signal                   :",
        row_631f[
            "capped_signal"
        ],
    )

    for path_631f, value_631f in row_631f[
        "target_fields"
    ][
        :30
    ]:

        print(
            f"    {path_631f} = {value_631f}"
        )


print(
    "\nFD003 historical source hits      :",
    len(
        source_target_hits_631f[
            "FD003"
        ]
    ),
)

for idx_631f, row_631f in enumerate(
    source_target_hits_631f[
        "FD003"
    ][
        :6
    ],
    start=1,
):

    print(
        f"\nFD003 SOURCE {idx_631f}: "
        f"{row_631f['origin']}"
    )

    print(
        "  raw / capped                    :",
        (
            row_631f[
                "raw_signal"
            ],
            row_631f[
                "capped_signal"
            ],
        ),
    )

    for context_631f in row_631f[
        "contexts"
    ][
        :4
    ]:

        print()

        for line_no_631f, line_631f in context_631f:

            print(
                f"    "
                f"{line_no_631f:05d} | "
                f"{line_631f}"
            )


print(
    "\nFD003 RESOLVED TARGET CLASS       :",
    fd003_target_class_631f,
)


print("\n--- FD004 TARGET-PROVENANCE CANDIDATES ---")

print(
    "Structured candidates             :",
    len(
        fd004_target_candidates_631f
    ),
)

for idx_631f, row_631f in enumerate(
    fd004_target_candidates_631f[
        :10
    ],
    start=1,
):

    print(
        f"\nFD004 STRUCTURED {idx_631f}: "
        f"{row_631f['name']}"
    )

    print(
        "  score                           :",
        row_631f[
            "score"
        ],
    )

    print(
        "  raw signal                      :",
        row_631f[
            "raw_signal"
        ],
    )

    print(
        "  capped signal                   :",
        row_631f[
            "capped_signal"
        ],
    )

    for path_631f, value_631f in row_631f[
        "target_fields"
    ][
        :30
    ]:

        print(
            f"    {path_631f} = {value_631f}"
        )


print(
    "\nFD004 historical source hits      :",
    len(
        source_target_hits_631f[
            "FD004"
        ]
    ),
)

for idx_631f, row_631f in enumerate(
    source_target_hits_631f[
        "FD004"
    ][
        :6
    ],
    start=1,
):

    print(
        f"\nFD004 SOURCE {idx_631f}: "
        f"{row_631f['origin']}"
    )

    print(
        "  raw / capped                    :",
        (
            row_631f[
                "raw_signal"
            ],
            row_631f[
                "capped_signal"
            ],
        ),
    )

    for context_631f in row_631f[
        "contexts"
    ][
        :4
    ]:

        print()

        for line_no_631f, line_631f in context_631f:

            print(
                f"    "
                f"{line_no_631f:05d} | "
                f"{line_631f}"
            )


print(
    "\nFD004 RESOLVED TARGET CLASS       :",
    fd004_target_class_631f,
)


print("\n--- FINAL TARGET-PROVENANCE STATUS ---")

print(
    "FD001                             :",
    "capped primary + raw secondary available",
)

print(
    "FD002                             :",
    "capped final authority only; no genuine raw final authority recovered",
)

print(
    "FD003                             :",
    fd003_target_class_631f,
)

print(
    "FD004                             :",
    fd004_target_class_631f,
)

print(
    "Target-provenance SHA-256         :",
    FD004_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631F,
)

print(
    "Closure audit SHA-256             :",
    FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F,
)


print(
    "\nPASS — FD002's apparent RAW-RUL authority was rejected as "
    "a recent-output false positive."
)

print(
    "No unsupported FD002 raw metric was created or inferred."
)

print(
    "Next: use the resolved target provenance to build the final "
    "FD001–FD004 synthesis with target-specific held-out results and "
    "only genuinely comparable quantities placed in the same comparison."
)

In [ ]:
# STEP 6.31G — FD003 / FD004 EXACT TARGET-AUTHORITY DISCRIMINATION
#
# Purpose:
# Resolve FD003 and FD004 target conventions using ONLY their own frozen
# scientific authorities, not broad global/source searches that can be
# contaminated by FD001/FD002 or recent synthesis text.
#
# FD003:
# - inspect only frozen FD003 closure / held-out score / pretest contracts
#
# FD004:
# - inspect only the frozen FD004 TRAIN-derived RUL contract and final closure
#
# REPORTING / PROVENANCE ONLY.
#
# NO inference.
# NO TEST/RUL access.
# NO metric recomputation.
# NO threshold/policy/model changes.
# NO cross-dataset synthesis table yet.

import re
import json
import hashlib
import numbers


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631g = (
    "all_checks_passed_631f",

    "FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_631F",
    "FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F",

    # FD003 frozen authorities
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "FD003_PRETEST_EVIDENCE_CLOSURE_629AX",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
    "FD003_DEV_STAGE_CONCLUSION_629AX",

    # FD004 frozen target / closure authorities
    "FD004_RUL_VARIABILITY_CONTRACT_630D",
    "FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
)

missing_631g = [
    name
    for name in required_631g
    if name not in globals()
]

if missing_631g:
    raise RuntimeError(
        "Missing STEP 6.31G authority object(s): "
        + ", ".join(missing_631g)
    )

if not bool(all_checks_passed_631f):
    raise RuntimeError(
        "STEP 6.31F must remain passed."
    )


def _json_sha256_631g(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631g(
        FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_631F
    )
    !=
    str(
        FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F
    )
):
    raise RuntimeError(
        "STEP 6.31F audit changed."
    )


# =============================================================================
# 1) GENERIC SCALAR FLATTENER
# =============================================================================

def _flatten_631g(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 12:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _flatten_631g(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) <= 500:
            for idx, child in enumerate(
                value
            ):
                _flatten_631g(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        output.append(
            (
                ".".join(path),
                value,
            )
        )

    return output


target_evidence_re_631g = re.compile(
    r"target|"
    r"rul|"
    r"cap|"
    r"clip|"
    r"uncapped|"
    r"piecewise",
    flags=re.IGNORECASE,
)


def _target_fields_631g(
    value,
):
    return tuple(
        (
            path,
            scalar,
        )
        for path, scalar
        in _flatten_631g(
            value
        )
        if target_evidence_re_631g.search(
            path
            +
            " "
            +
            str(scalar)
        )
    )


# =============================================================================
# 2) FD003 — DATASET-SPECIFIC FROZEN AUTHORITIES ONLY
# =============================================================================

fd003_authorities_631g = {
    "final_dataset_closure":
        FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,

    "final_scientific_conclusion":
        FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,

    "heldout_test_score":
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,

    "pretest_evidence_closure":
        FD003_PRETEST_EVIDENCE_CLOSURE_629AX,

    "fixed_policy_reporting":
        FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW,

    "dev_stage_conclusion":
        FD003_DEV_STAGE_CONCLUSION_629AX,
}


fd003_target_fields_631g = {}

for label_631g, obj_631g in fd003_authorities_631g.items():
    fd003_target_fields_631g[
        label_631g
    ] = _target_fields_631g(
        obj_631g
    )


fd003_target_text_631g = "\n".join(
    (
        authority_631g
        +
        "."
        +
        path_631g
        +
        "="
        +
        str(value_631g)
    )
    for authority_631g, rows_631g
    in fd003_target_fields_631g.items()
    for path_631g, value_631g
    in rows_631g
).lower()


# Strong signals only.
fd003_explicit_uncapped_631g = bool(
    re.search(
        r"uncapped|"
        r"rul_raw|"
        r"raw[_ ]?rul|"
        r"no[_ ]?cap|"
        r"no[_ ]?clip|"
        r"without[_ ]?cap",
        fd003_target_text_631g,
        flags=re.IGNORECASE,
    )
)

fd003_explicit_capped_631g = bool(
    re.search(
        r"rul_cap|"
        r"cap[_ ]?125|"
        r"piecewise[-_ ]?capped|"
        r"capped[_ ]?rul|"
        r"clip.*125",
        fd003_target_text_631g,
        flags=re.IGNORECASE,
    )
)


# =============================================================================
# 3) FD003 — TARGET NUMERIC SEMANTICS AUDIT
#
# Do NOT recompute metrics.
# Inspect only frozen contract descriptions/metadata.
# =============================================================================

fd003_rul_definition_candidates_631g = tuple(
    (
        authority_631g,
        path_631g,
        value_631g,
    )
    for authority_631g, rows_631g
    in fd003_target_fields_631g.items()
    for path_631g, value_631g
    in rows_631g
    if any(
        token in (
            path_631g
            +
            " "
            +
            str(value_631g)
        ).lower()
        for token in (
            "target",
            "rul_definition",
            "evaluation_target",
            "official_rul",
            "rul_source",
            "label",
            "cap",
            "clip",
            "uncapped",
        )
    )
)


# =============================================================================
# 4) FD004 — VERIFY EXACT TRAIN-DERIVED TARGET CONTRACT
# =============================================================================

if (
    _json_sha256_631g(
        FD004_RUL_VARIABILITY_CONTRACT_630D
    )
    !=
    str(
        FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D
    )
):
    raise RuntimeError(
        "FD004 frozen RUL/variability contract changed."
    )


fd004_rul_contract_631g = (
    FD004_RUL_VARIABILITY_CONTRACT_630D
)


required_fd004_target_keys_631g = (
    "rul_definition",
    "rul_source",
)

missing_fd004_target_keys_631g = [
    key
    for key in required_fd004_target_keys_631g
    if key not in fd004_rul_contract_631g
]

if missing_fd004_target_keys_631g:
    raise RuntimeError(
        "FD004 RUL contract missing target key(s): "
        + ", ".join(missing_fd004_target_keys_631g)
    )


fd004_rul_definition_631g = str(
    fd004_rul_contract_631g[
        "rul_definition"
    ]
)

fd004_rul_source_631g = str(
    fd004_rul_contract_631g[
        "rul_source"
    ]
)


fd004_explicit_uncapped_631g = bool(
    "uncapped"
    in fd004_rul_definition_631g.lower()
)

fd004_explicit_capped_631g = bool(
    re.search(
        r"capped|cap[_ ]?125|clip",
        fd004_rul_definition_631g,
        flags=re.IGNORECASE,
    )
)


if not fd004_explicit_uncapped_631g:
    raise RuntimeError(
        "FD004 frozen RUL contract no longer explicitly says uncapped."
    )

if fd004_explicit_capped_631g:
    raise RuntimeError(
        "FD004 frozen RUL definition unexpectedly contains capped semantics."
    )


# =============================================================================
# 5) FD004 FINAL-CLOSURE TARGET EVIDENCE
# =============================================================================

fd004_final_target_fields_631g = {
    "final_dataset_closure":
        _target_fields_631g(
            FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11
        ),

    "final_scientific_conclusion":
        _target_fields_631g(
            FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11
        ),
}


# =============================================================================
# 6) RESOLVE FD003 CONSERVATIVELY
#
# Do not infer "raw" merely from absence of cap terminology.
# =============================================================================

if (
    fd003_explicit_uncapped_631g
    and
    not fd003_explicit_capped_631g
):
    fd003_resolved_target_631g = (
        "RAW_OR_UNCAPPED_RUL_EXPLICIT"
    )

elif (
    fd003_explicit_capped_631g
    and
    not fd003_explicit_uncapped_631g
):
    fd003_resolved_target_631g = (
        "CAPPED_RUL_EXPLICIT"
    )

elif (
    fd003_explicit_uncapped_631g
    and
    fd003_explicit_capped_631g
):
    fd003_resolved_target_631g = (
        "MIXED_WITHIN_FD003_FROZEN_AUTHORITIES"
    )

else:
    fd003_resolved_target_631g = (
        "TARGET_CONVENTION_NOT_EXPLICIT_IN_FINAL_FROZEN_AUTHORITIES"
    )


# FD004 is explicit by frozen target-construction contract.
fd004_resolved_target_631g = (
    "RAW_OR_UNCAPPED_RUL_EXPLICIT"
)


# =============================================================================
# 7) FREEZE EXACT TARGET-PROVENANCE CONTRACT
# =============================================================================

FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G = {
    "FD001": {
        "status":
            "CAPPED_PRIMARY_AND_RAW_SECONDARY_AVAILABLE",

        "authority":
            FD001_FINAL_REPORTING_AUTHORITY_631D[
                "path"
            ],
    },

    "FD002": {
        "status":
            "CAPPED_ONLY_FINAL_HELDOUT_AUTHORITY",

        "evaluation_target":
            FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                "evaluation_target"
            ],

        "raw_final_authority_available":
            False,
    },

    "FD003": {
        "resolved_target_class":
            fd003_resolved_target_631g,

        "explicit_uncapped_signal":
            fd003_explicit_uncapped_631g,

        "explicit_capped_signal":
            fd003_explicit_capped_631g,

        "dataset_specific_authorities_inspected":
            tuple(
                fd003_authorities_631g.keys()
            ),
    },

    "FD004": {
        "resolved_target_class":
            fd004_resolved_target_631g,

        "rul_definition":
            fd004_rul_definition_631g,

        "rul_source":
            fd004_rul_source_631g,

        "primary_test_scored":
            False,
    },

    "comparison_rule":
        (
            "Absolute MAE comparisons require matching target conventions. "
            "Target-specific held-out outcomes may be reported side-by-side "
            "only with target convention made explicit."
        ),

    "broad_cross_dataset_search_used_for_resolution":
        False,
}


FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G = (
    _json_sha256_631g(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G
    )
)


audit_payload_631g = {
    "stage":
        "6.31G",

    "status":
        "FD003_FD004_EXACT_TARGET_AUTHORITY_DISCRIMINATION_COMPLETE",

    "fd003_target_class":
        fd003_resolved_target_631g,

    "fd003_explicit_uncapped_signal":
        fd003_explicit_uncapped_631g,

    "fd003_explicit_capped_signal":
        fd003_explicit_capped_631g,

    "fd004_target_class":
        fd004_resolved_target_631g,

    "fd004_rul_definition":
        fd004_rul_definition_631g,

    "cross_dataset_table_constructed":
        False,

    "scientific_recomputation":
        False,

    "test_rul_access":
        False,

    "model_inference":
        False,

    "threshold_policy_change":
        False,

    "reporting_only":
        True,
}


FD004_EXACT_TARGET_AUTHORITY_AUDIT_631G = (
    audit_payload_631g
)

FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G = (
    _json_sha256_631g(
        audit_payload_631g
    )
)

all_checks_passed_631g = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31G — FD003 / FD004 EXACT "
    "TARGET-AUTHORITY DISCRIMINATION"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("TEST/RUL access                   :", False)
print("Model inference                   :", False)
print("Threshold/policy change           :", False)
print("Broad global provenance search    :", False)
print("Cross-dataset table built         :", False)
print("Reporting-only                    :", True)


print("\n--- FD003 DATASET-SPECIFIC TARGET EVIDENCE ---")

print(
    "Frozen authorities inspected      :",
    tuple(
        fd003_authorities_631g.keys()
    ),
)

print(
    "Explicit uncapped/raw signal      :",
    fd003_explicit_uncapped_631g,
)

print(
    "Explicit capped signal            :",
    fd003_explicit_capped_631g,
)

print(
    "Resolved FD003 target class       :",
    fd003_resolved_target_631g,
)


print("\nFD003 target-related frozen fields:")

for (
    authority_631g,
    path_631g,
    value_631g,
) in fd003_rul_definition_candidates_631g[
    :100
]:

    print(
        f"  {authority_631g}.{path_631g} = "
        f"{value_631g}"
    )


print("\n--- FD004 EXACT FROZEN TARGET AUTHORITY ---")

print(
    "RUL definition                    :",
    fd004_rul_definition_631g,
)

print(
    "RUL source                        :",
    fd004_rul_source_631g,
)

print(
    "Explicit uncapped signal          :",
    fd004_explicit_uncapped_631g,
)

print(
    "Explicit capped signal            :",
    fd004_explicit_capped_631g,
)

print(
    "Resolved FD004 target class       :",
    fd004_resolved_target_631g,
)

print(
    "Primary held-out TEST scored      :",
    False,
)


print("\n--- FINAL TARGET-PROVENANCE MAP ---")

print(
    "FD001                             :",
    "capped primary + raw secondary",
)

print(
    "FD002                             :",
    "piecewise-capped held-out authority only",
)

print(
    "FD003                             :",
    fd003_resolved_target_631g,
)

print(
    "FD004                             :",
    fd004_resolved_target_631g,
)


print(
    "\nExact provenance SHA-256          :",
    FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G,
)

print(
    "Audit SHA-256                     :",
    FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G,
)


print(
    "\nPASS — FD003/FD004 target provenance was evaluated only from "
    "dataset-specific frozen scientific authorities."
)

print(
    "No recent cross-dataset synthesis object was allowed to determine "
    "either target convention."
)

print(
    "Next: if FD003 resolves explicitly, build the final target-aware "
    "FD001–FD004 synthesis table; otherwise perform one final FD003-only "
    "historical target-source recovery before synthesis."
)

In [ ]:
# STEP 6.31G-R1 — TARGET-CLASSIFIER REPAIR + EXACT PROVENANCE CLOSURE
#
# Repair-only step.
#
# Cause of 6.31G failure:
#     re.search(r"capped|...", "uncapped ...")
# returns True because "capped" is a substring of "uncapped".
#
# This cell:
# - re-certifies the frozen authorities
# - uses token-aware RAW/CAPPED classification
# - resolves FD003 from FD003-only frozen authorities
# - resolves FD004 from its exact frozen RUL construction contract
# - freezes the intended 6.31G provenance objects
#
# NO inference.
# NO TEST/RUL access.
# NO metric recomputation.
# NO threshold/policy/model changes.
# NO cross-dataset numerical synthesis yet.

import re
import json
import hashlib
import numbers


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631gr1 = (
    "all_checks_passed_631f",

    "FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_631F",
    "FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F",

    "FD001_FINAL_REPORTING_AUTHORITY_631D",
    "FD002_CAPPED_RUL_FINAL_AUTHORITY_631E",

    # FD003 frozen authorities
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "FD003_PRETEST_EVIDENCE_CLOSURE_629AX",
    "FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW",
    "FD003_DEV_STAGE_CONCLUSION_629AX",

    # FD004 frozen authorities
    "FD004_RUL_VARIABILITY_CONTRACT_630D",
    "FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
)

missing_631gr1 = [
    name
    for name in required_631gr1
    if name not in globals()
]

if missing_631gr1:
    raise RuntimeError(
        "Missing STEP 6.31G-R1 authority object(s): "
        + ", ".join(missing_631gr1)
    )

if not bool(all_checks_passed_631f):
    raise RuntimeError(
        "STEP 6.31F must remain passed."
    )


def _json_sha256_631gr1(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631gr1(
        FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_631F
    )
    !=
    str(
        FD004_TARGET_PROVENANCE_CLOSURE_AUDIT_SHA256_631F
    )
):
    raise RuntimeError(
        "STEP 6.31F audit changed."
    )


# =============================================================================
# 1) GENERIC SCALAR FLATTENER
# =============================================================================

def _flatten_631gr1(
    value,
    path=(),
    depth=0,
    output=None,
):
    if output is None:
        output = []

    if depth > 12:
        return output

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _flatten_631gr1(
                child,
                path + (str(key),),
                depth + 1,
                output,
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        if len(value) <= 500:
            for idx, child in enumerate(
                value
            ):
                _flatten_631gr1(
                    child,
                    path + (f"[{idx}]",),
                    depth + 1,
                    output,
                )

    elif isinstance(
        value,
        (
            str,
            bool,
            numbers.Number,
            type(None),
        ),
    ):
        output.append(
            (
                ".".join(path),
                value,
            )
        )

    return output


target_field_re_631gr1 = re.compile(
    r"target|"
    r"rul|"
    r"cap|"
    r"clip|"
    r"uncapped|"
    r"piecewise",
    flags=re.IGNORECASE,
)


def _target_fields_631gr1(
    value,
):
    return tuple(
        (
            path,
            scalar,
        )
        for path, scalar
        in _flatten_631gr1(
            value
        )
        if target_field_re_631gr1.search(
            path
            +
            " "
            +
            str(scalar)
        )
    )


# =============================================================================
# 2) TOKEN-AWARE TARGET CLASSIFIER
#
# Critical repair:
# "uncapped" MUST NOT trigger the capped detector.
# =============================================================================

raw_target_re_631gr1 = re.compile(
    r"\buncapped\b|"
    r"\brul_raw\b|"
    r"\braw_rul\b|"
    r"\braw\s+rul\b|"
    r"\bno[_\s-]?cap(?:ping)?\b|"
    r"\bno[_\s-]?clip(?:ping)?\b|"
    r"\bwithout[_\s-]?cap(?:ping)?\b",
    flags=re.IGNORECASE,
)


capped_target_re_631gr1 = re.compile(
    r"\brul_cap(?:_?\d+)?\b|"
    r"\bcap[_\s-]?125\b|"
    r"\bpiecewise[-_\s]?capped\b|"
    r"(?<!un)\bcapped\b|"
    r"\bcapped[_\s-]?rul\b|"
    r"\bclip(?:ped|ping)?[_\s-]?(?:at[_\s-]?)?125\b",
    flags=re.IGNORECASE,
)


def _classify_target_text_631gr1(
    text,
):
    raw = bool(
        raw_target_re_631gr1.search(
            text
        )
    )

    capped = bool(
        capped_target_re_631gr1.search(
            text
        )
    )

    if raw and not capped:
        resolved = "RAW_OR_UNCAPPED_RUL_EXPLICIT"

    elif capped and not raw:
        resolved = "CAPPED_RUL_EXPLICIT"

    elif raw and capped:
        resolved = "BOTH_RAW_AND_CAPPED_EXPLICIT"

    else:
        resolved = "TARGET_CONVENTION_NOT_EXPLICIT"

    return (
        resolved,
        raw,
        capped,
    )


# =============================================================================
# 3) FD003 — INSPECT ONLY FROZEN FD003 AUTHORITIES
# =============================================================================

fd003_authorities_631gr1 = {
    "final_dataset_closure":
        FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC,

    "final_scientific_conclusion":
        FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,

    "heldout_test_score":
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,

    "pretest_evidence_closure":
        FD003_PRETEST_EVIDENCE_CLOSURE_629AX,

    "fixed_policy_reporting":
        FD003_FIXED_POLICY_REPORTING_CONTRACT_629AW,

    "dev_stage_conclusion":
        FD003_DEV_STAGE_CONCLUSION_629AX,
}


fd003_target_fields_631gr1 = {
    label:
        _target_fields_631gr1(
            obj
        )
    for label, obj
    in fd003_authorities_631gr1.items()
}


fd003_target_text_631gr1 = "\n".join(
    (
        authority
        +
        "."
        +
        path
        +
        "="
        +
        str(value)
    )
    for authority, rows
    in fd003_target_fields_631gr1.items()
    for path, value
    in rows
)


(
    fd003_resolved_target_631gr1,
    fd003_explicit_raw_631gr1,
    fd003_explicit_capped_631gr1,
) = _classify_target_text_631gr1(
    fd003_target_text_631gr1
)


fd003_decisive_target_fields_631gr1 = tuple(
    (
        authority,
        path,
        value,
    )
    for authority, rows
    in fd003_target_fields_631gr1.items()
    for path, value
    in rows
    if (
        raw_target_re_631gr1.search(
            path
            +
            " "
            +
            str(value)
        )
        or
        capped_target_re_631gr1.search(
            path
            +
            " "
            +
            str(value)
        )
    )
)


# =============================================================================
# 4) FD004 — EXACT FROZEN TARGET-CONSTRUCTION AUTHORITY
# =============================================================================

if (
    _json_sha256_631gr1(
        FD004_RUL_VARIABILITY_CONTRACT_630D
    )
    !=
    str(
        FD004_RUL_VARIABILITY_CONTRACT_SHA256_630D
    )
):
    raise RuntimeError(
        "FD004 frozen RUL/variability contract changed."
    )


fd004_rul_contract_631gr1 = (
    FD004_RUL_VARIABILITY_CONTRACT_630D
)


fd004_rul_definition_631gr1 = str(
    fd004_rul_contract_631gr1[
        "rul_definition"
    ]
)

fd004_rul_source_631gr1 = str(
    fd004_rul_contract_631gr1[
        "rul_source"
    ]
)


(
    fd004_resolved_target_631gr1,
    fd004_explicit_raw_631gr1,
    fd004_explicit_capped_631gr1,
) = _classify_target_text_631gr1(
    fd004_rul_definition_631gr1
)


if (
    fd004_resolved_target_631gr1
    !=
    "RAW_OR_UNCAPPED_RUL_EXPLICIT"
):
    raise RuntimeError(
        "FD004 frozen target did not resolve to explicit uncapped RUL."
    )


if not fd004_explicit_raw_631gr1:
    raise RuntimeError(
        "FD004 uncapped target signal is missing."
    )


if fd004_explicit_capped_631gr1:
    raise RuntimeError(
        "FD004 token-aware classifier still found genuine capped semantics."
    )


# =============================================================================
# 5) DEMONSTRATE THE ORIGINAL BUG EXPLICITLY
# =============================================================================

original_bug_demo_631gr1 = {
    "rul_definition":
        fd004_rul_definition_631gr1,

    "naive_substring_capped":
        bool(
            re.search(
                r"capped",
                fd004_rul_definition_631gr1,
                flags=re.IGNORECASE,
            )
        ),

    "token_aware_capped":
        bool(
            capped_target_re_631gr1.search(
                fd004_rul_definition_631gr1
            )
        ),

    "explicit_uncapped":
        bool(
            raw_target_re_631gr1.search(
                fd004_rul_definition_631gr1
            )
        ),
}


if not original_bug_demo_631gr1[
    "naive_substring_capped"
]:
    raise RuntimeError(
        "Expected demonstration of the original substring bug did not occur."
    )

if original_bug_demo_631gr1[
    "token_aware_capped"
]:
    raise RuntimeError(
        "Token-aware capped detector incorrectly matches 'uncapped'."
    )

if not original_bug_demo_631gr1[
    "explicit_uncapped"
]:
    raise RuntimeError(
        "Token-aware raw detector failed to match 'uncapped'."
    )


# =============================================================================
# 6) FREEZE EXACT CROSS-DATASET TARGET PROVENANCE
# =============================================================================

FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G = {
    "FD001": {
        "status":
            "CAPPED_PRIMARY_AND_RAW_SECONDARY_AVAILABLE",

        "authority":
            FD001_FINAL_REPORTING_AUTHORITY_631D[
                "path"
            ],
    },

    "FD002": {
        "status":
            "CAPPED_ONLY_FINAL_HELDOUT_AUTHORITY",

        "evaluation_target":
            FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                "evaluation_target"
            ],

        "raw_final_authority_available":
            False,
    },

    "FD003": {
        "resolved_target_class":
            fd003_resolved_target_631gr1,

        "explicit_raw_or_uncapped_signal":
            fd003_explicit_raw_631gr1,

        "explicit_capped_signal":
            fd003_explicit_capped_631gr1,

        "dataset_specific_authorities_inspected":
            tuple(
                fd003_authorities_631gr1.keys()
            ),

        "decisive_target_fields":
            fd003_decisive_target_fields_631gr1,
    },

    "FD004": {
        "resolved_target_class":
            fd004_resolved_target_631gr1,

        "rul_definition":
            fd004_rul_definition_631gr1,

        "rul_source":
            fd004_rul_source_631gr1,

        "explicit_raw_or_uncapped_signal":
            fd004_explicit_raw_631gr1,

        "explicit_capped_signal":
            fd004_explicit_capped_631gr1,

        "primary_test_scored":
            False,
    },

    "comparison_rule":
        (
            "Absolute MAE comparisons require matching target conventions. "
            "Target-specific held-out outcomes may be shown side-by-side "
            "only with target convention explicitly identified."
        ),

    "fd002_raw_final_authority_available":
        False,

    "broad_cross_dataset_search_used_for_resolution":
        False,
}


FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G = (
    _json_sha256_631gr1(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G
    )
)


# =============================================================================
# 7) FREEZE REPAIR / CLOSURE AUDIT
# =============================================================================

FD004_EXACT_TARGET_AUTHORITY_AUDIT_631G = {
    "stage":
        "6.31G-R1",

    "status":
        "FD003_FD004_EXACT_TARGET_AUTHORITY_DISCRIMINATION_COMPLETE",

    "original_631g_failure":
        "TOKENIZATION_BUG_CAPPED_SUBSTRING_INSIDE_UNCAPPED",

    "original_failure_scientific":
        False,

    "fd003_target_class":
        fd003_resolved_target_631gr1,

    "fd003_explicit_raw_or_uncapped_signal":
        fd003_explicit_raw_631gr1,

    "fd003_explicit_capped_signal":
        fd003_explicit_capped_631gr1,

    "fd004_target_class":
        fd004_resolved_target_631gr1,

    "fd004_rul_definition":
        fd004_rul_definition_631gr1,

    "fd004_token_aware_capped_signal":
        fd004_explicit_capped_631gr1,

    "cross_dataset_table_constructed":
        False,

    "scientific_recomputation":
        False,

    "test_rul_access":
        False,

    "model_inference":
        False,

    "threshold_policy_change":
        False,

    "reporting_only":
        True,
}


FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G = (
    _json_sha256_631gr1(
        FD004_EXACT_TARGET_AUTHORITY_AUDIT_631G
    )
)


FD004_TARGET_CLASSIFIER_REPAIR_631GR1 = (
    original_bug_demo_631gr1
)

FD004_TARGET_CLASSIFIER_REPAIR_SHA256_631GR1 = (
    _json_sha256_631gr1(
        FD004_TARGET_CLASSIFIER_REPAIR_631GR1
    )
)


all_checks_passed_631g = True
all_checks_passed_631gr1 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31G-R1 — TARGET-CLASSIFIER REPAIR + "
    "EXACT PROVENANCE CLOSURE"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("TEST/RUL access                   :", False)
print("Model inference                   :", False)
print("Threshold/policy change           :", False)
print("Cross-dataset table built         :", False)
print("Reporting-only                    :", True)


print("\n--- ORIGINAL 6.31G FAILURE ---")

print(
    "FD004 RUL definition              :",
    fd004_rul_definition_631gr1,
)

print(
    "Naive substring 'capped' match    :",
    original_bug_demo_631gr1[
        "naive_substring_capped"
    ],
)

print(
    "Token-aware capped match          :",
    original_bug_demo_631gr1[
        "token_aware_capped"
    ],
)

print(
    "Token-aware uncapped match        :",
    original_bug_demo_631gr1[
        "explicit_uncapped"
    ],
)

print(
    "Failure classification            :",
    "technical classifier bug only",
)


print("\n--- FD003 EXACT DATASET-SPECIFIC TARGET EVIDENCE ---")

print(
    "Resolved target class             :",
    fd003_resolved_target_631gr1,
)

print(
    "Explicit raw/uncapped signal      :",
    fd003_explicit_raw_631gr1,
)

print(
    "Explicit capped signal            :",
    fd003_explicit_capped_631gr1,
)

print(
    "Decisive target fields            :",
    len(
        fd003_decisive_target_fields_631gr1
    ),
)

for (
    authority_631gr1,
    path_631gr1,
    value_631gr1,
) in fd003_decisive_target_fields_631gr1[
    :60
]:

    print(
        f"  {authority_631gr1}.{path_631gr1} = "
        f"{value_631gr1}"
    )


print("\n--- FD004 EXACT FROZEN TARGET AUTHORITY ---")

print(
    "RUL definition                    :",
    fd004_rul_definition_631gr1,
)

print(
    "RUL source                        :",
    fd004_rul_source_631gr1,
)

print(
    "Resolved target class             :",
    fd004_resolved_target_631gr1,
)

print(
    "Explicit raw/uncapped signal      :",
    fd004_explicit_raw_631gr1,
)

print(
    "Explicit capped signal            :",
    fd004_explicit_capped_631gr1,
)

print(
    "Primary held-out TEST scored      :",
    False,
)


print("\n--- FINAL TARGET-PROVENANCE MAP ---")

print(
    "FD001                             :",
    "capped primary + raw secondary available",
)

print(
    "FD002                             :",
    "piecewise-capped held-out authority only",
)

print(
    "FD003                             :",
    fd003_resolved_target_631gr1,
)

print(
    "FD004                             :",
    fd004_resolved_target_631gr1,
)


print(
    "\nTarget-classifier repair SHA-256  :",
    FD004_TARGET_CLASSIFIER_REPAIR_SHA256_631GR1,
)

print(
    "Exact provenance SHA-256          :",
    FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G,
)

print(
    "Audit SHA-256                     :",
    FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G,
)


print(
    "\nPASS — the 6.31G failure was a string-matching bug only."
)

print(
    "FD004 is now correctly classified from its frozen target contract "
    "without treating the word 'uncapped' as capped."
)

print(
    "Next: if FD003 is now explicitly resolved, construct the final "
    "target-aware FD001–FD004 synthesis. If FD003 remains non-explicit, "
    "perform one FD003-only source recovery before synthesis."
)

In [ ]:
# STEP 6.31H — FD003 HISTORICAL TARGET-CONSTRUCTION RECOVERY
#
# Purpose:
# Resolve the final remaining target-provenance ambiguity:
#     What exact RUL target convention governed FD003?
#
# Evidence scope:
#   1) FD003-specific notebook SOURCE cells only
#   2) FD003 checkpoint/report JSON text only
#
# Explicitly excluded:
#   - raw C-MAPSS TRAIN / TEST / RUL files
#   - model inference
#   - MC dropout
#   - metric recomputation
#   - any new label construction
#   - any threshold/policy/model changes
#
# This is SOURCE / PROVENANCE FORENSICS ONLY.

import re
import json
import hashlib
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631h = (
    "all_checks_passed_631gr1",

    "FD004_EXACT_TARGET_AUTHORITY_AUDIT_631G",
    "FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G",

    "FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G",
    "FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G",
)

missing_631h = [
    name
    for name in required_631h
    if name not in globals()
]

if missing_631h:
    raise RuntimeError(
        "Missing STEP 6.31H authority object(s): "
        + ", ".join(missing_631h)
    )

if not bool(all_checks_passed_631gr1):
    raise RuntimeError(
        "STEP 6.31G-R1 must remain passed."
    )


def _json_sha256_631h(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631h(
        FD004_EXACT_TARGET_AUTHORITY_AUDIT_631G
    )
    !=
    str(
        FD004_EXACT_TARGET_AUTHORITY_AUDIT_SHA256_631G
    )
):
    raise RuntimeError(
        "STEP 6.31G-R1 audit changed."
    )


if (
    _json_sha256_631h(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G
    )
    !=
    str(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G
    )
):
    raise RuntimeError(
        "STEP 6.31G-R1 target provenance changed."
    )


if (
    FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G[
        "FD003"
    ][
        "resolved_target_class"
    ]
    !=
    "TARGET_CONVENTION_NOT_EXPLICIT"
):
    raise RuntimeError(
        "FD003 no longer requires historical target recovery."
    )


# =============================================================================
# 1) TOKEN-AWARE TARGET SEMANTICS
# =============================================================================

raw_target_re_631h = re.compile(
    r"\buncapped\b|"
    r"\brul_raw\b|"
    r"\braw_rul\b|"
    r"\braw\s+rul\b|"
    r"\bterminal[_\s-]?cycle\s*-\s*current[_\s-]?cycle\b|"
    r"\bmax[_\s-]?cycle\s*-\s*cycle\b|"
    r"\bno[_\s-]?cap(?:ping)?\b|"
    r"\bno[_\s-]?clip(?:ping)?\b",
    flags=re.IGNORECASE,
)


capped_target_re_631h = re.compile(
    r"\brul_cap(?:_?\d+)?\b|"
    r"\bcap[_\s-]?125\b|"
    r"\bpiecewise[-_\s]?capped\b|"
    r"(?<!un)\bcapped\b|"
    r"\bcapped[_\s-]?rul\b|"
    r"\bclip(?:ped|ping)?[_\s-]?(?:at[_\s-]?)?\d+\b",
    flags=re.IGNORECASE,
)


construction_marker_re_631h = re.compile(
    r"FD003_.*RUL|"
    r"FIT_RUL|"
    r"DEV_RUL|"
    r"RUL_LABEL|"
    r"rul_definition|"
    r"terminal_cycle|"
    r"max_cycle|"
    r"current_cycle|"
    r"construct.*rul|"
    r"target.*rul",
    flags=re.IGNORECASE,
)


def _classify_text_631h(text):

    raw = bool(
        raw_target_re_631h.search(
            text
        )
    )

    capped = bool(
        capped_target_re_631h.search(
            text
        )
    )

    if raw and not capped:
        resolved = "RAW_OR_UNCAPPED_RUL_EXPLICIT"

    elif capped and not raw:
        resolved = "CAPPED_RUL_EXPLICIT"

    elif raw and capped:
        resolved = "BOTH_RAW_AND_CAPPED_EXPLICIT"

    else:
        resolved = "TARGET_CONVENTION_NOT_EXPLICIT"

    return (
        resolved,
        raw,
        capped,
    )


# =============================================================================
# 2) SEARCH FD003-SPECIFIC NOTEBOOK SOURCE ONLY
# =============================================================================

notebook_root_631h = Path(
    r"C:\RUL_Reliability_Research\03_notebooks"
)

if not notebook_root_631h.is_dir():
    raise RuntimeError(
        "Notebook directory was not found."
    )


notebook_paths_631h = sorted(
    notebook_root_631h.rglob(
        "*.ipynb"
    )
)


source_candidates_631h = []


for notebook_path_631h in notebook_paths_631h:

    try:
        notebook_obj_631h = json.loads(
            notebook_path_631h.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        continue


    for cell_index_631h, cell_631h in enumerate(
        notebook_obj_631h.get(
            "cells",
            []
        )
    ):

        if cell_631h.get(
            "cell_type"
        ) != "code":
            continue


        source_raw_631h = cell_631h.get(
            "source",
            []
        )

        if isinstance(
            source_raw_631h,
            list,
        ):
            source_631h = "".join(
                str(x)
                for x in source_raw_631h
            )
        else:
            source_631h = str(
                source_raw_631h
            )


        if not source_631h.strip():
            continue


        lower_source_631h = (
            source_631h.lower()
        )


        # ---------------------------------------------------------------------
        # Strict FD003 locality.
        # ---------------------------------------------------------------------

        if "fd003" not in lower_source_631h:
            continue


        # ---------------------------------------------------------------------
        # Exclude current synthesis/provenance cells.
        # ---------------------------------------------------------------------

        if any(
            token in lower_source_631h
            for token in (
                "step 6.31",
                "631a",
                "631b",
                "631c",
                "631d",
                "631e",
                "631f",
                "631g",
                "631h",
            )
        ):
            continue


        if not construction_marker_re_631h.search(
            source_631h
        ):
            continue


        (
            target_class_631h,
            raw_signal_631h,
            capped_signal_631h,
        ) = _classify_text_631h(
            source_631h
        )


        # ---------------------------------------------------------------------
        # Focus excerpt around target-construction semantics.
        # ---------------------------------------------------------------------

        lines_631h = source_631h.splitlines()

        match_indices_631h = [
            idx
            for idx, line
            in enumerate(
                lines_631h
            )
            if (
                construction_marker_re_631h.search(
                    line
                )
                or
                raw_target_re_631h.search(
                    line
                )
                or
                capped_target_re_631h.search(
                    line
                )
            )
        ]


        selected_indices_631h = set()

        for idx_631h in match_indices_631h:

            selected_indices_631h.update(
                range(
                    max(
                        0,
                        idx_631h - 5,
                    ),
                    min(
                        len(
                            lines_631h
                        ),
                        idx_631h + 9,
                    ),
                )
            )


        excerpt_631h = "\n".join(
            f"{idx_631h + 1:04d}: "
            f"{lines_631h[idx_631h]}"
            for idx_631h
            in sorted(
                selected_indices_631h
            )
        )


        score_631h = 0

        if raw_signal_631h:
            score_631h += 100

        if capped_signal_631h:
            score_631h += 100

        for token, weight in (
            ("FD003_FIT_RUL", 80),
            ("FD003_DEV_RUL", 80),
            ("rul_definition", 80),
            ("terminal_cycle", 60),
            ("max_cycle", 50),
            ("current_cycle", 50),
            ("uncapped", 70),
            ("RUL_cap", 70),
            ("cap_125", 70),
            ("clip", 40),
        ):
            if token.lower() in lower_source_631h:
                score_631h += weight


        source_candidates_631h.append(
            {
                "notebook":
                    str(
                        notebook_path_631h
                    ),

                "cell_index":
                    int(
                        cell_index_631h
                    ),

                "source_sha256":
                    hashlib.sha256(
                        source_631h.encode(
                            "utf-8"
                        )
                    ).hexdigest(),

                "target_class":
                    target_class_631h,

                "raw_signal":
                    raw_signal_631h,

                "capped_signal":
                    capped_signal_631h,

                "score":
                    int(
                        score_631h
                    ),

                "excerpt":
                    excerpt_631h,
            }
        )


source_candidates_631h = sorted(
    source_candidates_631h,
    key=lambda row: (
        -row[
            "score"
        ],
        row[
            "notebook"
        ],
        row[
            "cell_index"
        ],
    ),
)


# =============================================================================
# 3) SEARCH FD003 CHECKPOINT / REPORT JSON TEXT ONLY
# =============================================================================

project_root_631h = Path(
    r"C:\RUL_Reliability_Research"
)

json_roots_631h = (
    project_root_631h
    /
    "checkpoints",

    project_root_631h
    /
    "06_results",
)


json_candidates_631h = []


for json_root_631h in json_roots_631h:

    if not json_root_631h.is_dir():
        continue


    for path_631h in json_root_631h.rglob(
        "*.json"
    ):

        lower_path_631h = str(
            path_631h
        ).lower()

        if "fd003" not in lower_path_631h:
            continue


        try:
            size_631h = path_631h.stat().st_size
        except OSError:
            continue

        if size_631h > 10_000_000:
            continue


        try:
            text_631h = path_631h.read_text(
                encoding="utf-8"
            )
        except Exception:
            continue


        if not construction_marker_re_631h.search(
            text_631h
        ):
            continue


        (
            target_class_631h,
            raw_signal_631h,
            capped_signal_631h,
        ) = _classify_text_631h(
            text_631h
        )


        if not (
            raw_signal_631h
            or
            capped_signal_631h
        ):
            continue


        try:
            payload_631h = json.loads(
                text_631h
            )
        except Exception:
            payload_631h = None


        evidence_rows_631h = []


        def _walk_json_631h(
            value,
            path=(),
            depth=0,
        ):
            if depth > 10:
                return

            if isinstance(
                value,
                dict,
            ):

                for key, child in value.items():

                    _walk_json_631h(
                        child,
                        path + (str(key),),
                        depth + 1,
                    )

            elif isinstance(
                value,
                (
                    list,
                    tuple,
                ),
            ):

                if len(value) <= 500:

                    for idx, child in enumerate(
                        value
                    ):

                        _walk_json_631h(
                            child,
                            path + (f"[{idx}]",),
                            depth + 1,
                        )

            else:

                scalar_text_631h = str(
                    value
                )

                path_text_631h = ".".join(
                    path
                )

                combined_631h = (
                    path_text_631h
                    +
                    " "
                    +
                    scalar_text_631h
                )

                if (
                    construction_marker_re_631h.search(
                        combined_631h
                    )
                    or
                    raw_target_re_631h.search(
                        combined_631h
                    )
                    or
                    capped_target_re_631h.search(
                        combined_631h
                    )
                ):

                    evidence_rows_631h.append(
                        (
                            path_text_631h,
                            value,
                        )
                    )


        if payload_631h is not None:

            _walk_json_631h(
                payload_631h
            )


        json_candidates_631h.append(
            {
                "path":
                    str(
                        path_631h
                    ),

                "file_sha256":
                    hashlib.sha256(
                        path_631h.read_bytes()
                    ).hexdigest(),

                "target_class":
                    target_class_631h,

                "raw_signal":
                    raw_signal_631h,

                "capped_signal":
                    capped_signal_631h,

                "evidence_rows":
                    tuple(
                        evidence_rows_631h[
                            :100
                        ]
                    ),
            }
        )


# =============================================================================
# 4) RESOLVE ONLY FROM DECISIVE FD003-SPECIFIC EVIDENCE
# =============================================================================

decisive_raw_candidates_631h = tuple(
    row
    for row in source_candidates_631h
    if (
        row[
            "raw_signal"
        ]
        and
        not row[
            "capped_signal"
        ]
    )
)


decisive_capped_candidates_631h = tuple(
    row
    for row in source_candidates_631h
    if (
        row[
            "capped_signal"
        ]
        and
        not row[
            "raw_signal"
        ]
    )
)


json_raw_candidates_631h = tuple(
    row
    for row in json_candidates_631h
    if (
        row[
            "raw_signal"
        ]
        and
        not row[
            "capped_signal"
        ]
    )
)


json_capped_candidates_631h = tuple(
    row
    for row in json_candidates_631h
    if (
        row[
            "capped_signal"
        ]
        and
        not row[
            "raw_signal"
        ]
    )
)


raw_evidence_exists_631h = bool(
    decisive_raw_candidates_631h
    or
    json_raw_candidates_631h
)

capped_evidence_exists_631h = bool(
    decisive_capped_candidates_631h
    or
    json_capped_candidates_631h
)


if (
    raw_evidence_exists_631h
    and
    not capped_evidence_exists_631h
):
    FD003_RESOLVED_TARGET_CLASS_631H = (
        "RAW_OR_UNCAPPED_RUL_EXPLICIT"
    )

elif (
    capped_evidence_exists_631h
    and
    not raw_evidence_exists_631h
):
    FD003_RESOLVED_TARGET_CLASS_631H = (
        "CAPPED_RUL_EXPLICIT"
    )

elif (
    raw_evidence_exists_631h
    and
    capped_evidence_exists_631h
):
    FD003_RESOLVED_TARGET_CLASS_631H = (
        "CONFLICTING_FD003_TARGET_EVIDENCE_REQUIRES_MANUAL_DISCRIMINATION"
    )

else:
    FD003_RESOLVED_TARGET_CLASS_631H = (
        "NO_EXPLICIT_HISTORICAL_TARGET_CONSTRUCTION_RECOVERED"
    )


# =============================================================================
# 5) FREEZE RECOVERY CONTRACT
# =============================================================================

FD003_TARGET_CONSTRUCTION_RECOVERY_631H = {
    "stage":
        "6.31H",

    "dataset":
        "FD003",

    "resolved_target_class":
        FD003_RESOLVED_TARGET_CLASS_631H,

    "source_candidate_count":
        len(
            source_candidates_631h
        ),

    "json_candidate_count":
        len(
            json_candidates_631h
        ),

    "decisive_raw_source_count":
        len(
            decisive_raw_candidates_631h
        ),

    "decisive_capped_source_count":
        len(
            decisive_capped_candidates_631h
        ),

    "decisive_raw_json_count":
        len(
            json_raw_candidates_631h
        ),

    "decisive_capped_json_count":
        len(
            json_capped_candidates_631h
        ),

    "raw_evidence_exists":
        raw_evidence_exists_631h,

    "capped_evidence_exists":
        capped_evidence_exists_631h,

    "top_source_candidates":
        tuple(
            {
                "notebook":
                    row[
                        "notebook"
                    ],

                "cell_index":
                    row[
                        "cell_index"
                    ],

                "source_sha256":
                    row[
                        "source_sha256"
                    ],

                "target_class":
                    row[
                        "target_class"
                    ],

                "raw_signal":
                    row[
                        "raw_signal"
                    ],

                "capped_signal":
                    row[
                        "capped_signal"
                    ],

                "score":
                    row[
                        "score"
                    ],
            }
            for row
            in source_candidates_631h[
                :15
            ]
        ),

    "json_candidates":
        tuple(
            {
                "path":
                    row[
                        "path"
                    ],

                "file_sha256":
                    row[
                        "file_sha256"
                    ],

                "target_class":
                    row[
                        "target_class"
                    ],

                "raw_signal":
                    row[
                        "raw_signal"
                    ],

                "capped_signal":
                    row[
                        "capped_signal"
                    ],
            }
            for row
            in json_candidates_631h[
                :15
            ]
        ),

    "raw_dataset_files_read":
        False,

    "historical_cells_executed":
        False,

    "labels_recomputed":
        False,

    "metrics_recomputed":
        False,

    "reporting_only":
        True,
}


FD003_TARGET_CONSTRUCTION_RECOVERY_SHA256_631H = (
    _json_sha256_631h(
        FD003_TARGET_CONSTRUCTION_RECOVERY_631H
    )
)


all_checks_passed_631h = True


# =============================================================================
# 6) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31H — FD003 HISTORICAL "
    "TARGET-CONSTRUCTION RECOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Historical cells executed         :", False)
print("Raw C-MAPSS files read            :", False)
print("Labels recomputed                 :", False)
print("Metrics recomputed                :", False)
print("Model inference                   :", False)
print("Threshold/policy change           :", False)
print("Reporting/provenance only         :", True)


print("\n--- FD003 NOTEBOOK SOURCE CANDIDATES ---")

print(
    "Candidate count                   :",
    len(
        source_candidates_631h
    ),
)


for idx_631h, row_631h in enumerate(
    source_candidates_631h[
        :12
    ],
    start=1,
):

    print(
        f"\nSOURCE {idx_631h}"
    )

    print(
        "  notebook                        :",
        row_631h[
            "notebook"
        ],
    )

    print(
        "  cell index                      :",
        row_631h[
            "cell_index"
        ],
    )

    print(
        "  score                           :",
        row_631h[
            "score"
        ],
    )

    print(
        "  target class                    :",
        row_631h[
            "target_class"
        ],
    )

    print(
        "  raw / capped                    :",
        (
            row_631h[
                "raw_signal"
            ],
            row_631h[
                "capped_signal"
            ],
        ),
    )

    print(
        "  source SHA-256                  :",
        row_631h[
            "source_sha256"
        ],
    )

    print(
        "\n  TARGET-CONSTRUCTION EXCERPT:\n"
    )

    print(
        row_631h[
            "excerpt"
        ][
            :10000
        ]
    )


print("\n--- FD003 CHECKPOINT / REPORT JSON CANDIDATES ---")

print(
    "Candidate count                   :",
    len(
        json_candidates_631h
    ),
)


for idx_631h, row_631h in enumerate(
    json_candidates_631h[
        :12
    ],
    start=1,
):

    print(
        f"\nJSON {idx_631h}"
    )

    print(
        "  path                            :",
        row_631h[
            "path"
        ],
    )

    print(
        "  target class                    :",
        row_631h[
            "target_class"
        ],
    )

    print(
        "  raw / capped                    :",
        (
            row_631h[
                "raw_signal"
            ],
            row_631h[
                "capped_signal"
            ],
        ),
    )

    print(
        "  file SHA-256                    :",
        row_631h[
            "file_sha256"
        ],
    )

    for path_631h, value_631h in row_631h[
        "evidence_rows"
    ][
        :50
    ]:

        print(
            f"    {path_631h} = "
            f"{value_631h}"
        )


print("\n--- DECISIVE EVIDENCE COUNTS ---")

print(
    "Raw-only notebook sources         :",
    len(
        decisive_raw_candidates_631h
    ),
)

print(
    "Capped-only notebook sources      :",
    len(
        decisive_capped_candidates_631h
    ),
)

print(
    "Raw-only JSON authorities         :",
    len(
        json_raw_candidates_631h
    ),
)

print(
    "Capped-only JSON authorities      :",
    len(
        json_capped_candidates_631h
    ),
)


print("\n--- FD003 FINAL TARGET RESOLUTION ---")

print(
    "Raw evidence exists               :",
    raw_evidence_exists_631h,
)

print(
    "Capped evidence exists            :",
    capped_evidence_exists_631h,
)

print(
    "Resolved target class             :",
    FD003_RESOLVED_TARGET_CLASS_631H,
)

print(
    "Recovery SHA-256                  :",
    FD003_TARGET_CONSTRUCTION_RECOVERY_SHA256_631H,
)


print(
    "\nPASS — FD003 target provenance was searched using FD003-specific "
    "historical source/checkpoint evidence only."
)

print(
    "No target convention was inferred merely from missing terminology."
)

print(
    "Next: if this resolves to exactly one target convention, freeze the "
    "final target-aware FD001–FD004 synthesis table from already-locked "
    "results only."
)

In [ ]:
# STEP 6.31I — FINAL TARGET-AWARE FD001–FD004 SYNTHESIS LOCK
#
# Purpose:
# Build the final cross-dataset scientific synthesis from ALREADY-FROZEN
# aggregate evidence only, now that target provenance is resolved:
#
#   FD001 : capped primary + raw secondary held-out metrics
#   FD002 : piecewise-capped held-out metrics only
#   FD003 : raw/uncapped held-out metrics
#   FD004 : raw/uncapped; primary held-out TEST structurally UNSCORED
#
# Also retain FD004 clean-DEV / controlled-corruption evidence separately
# from held-out TEST evidence.
#
# IMPORTANT:
# - No predictions are regenerated.
# - No labels are reconstructed.
# - No raw TEST/RUL files are read.
# - No model inference / MC dropout.
# - No threshold or policy selection.
# - No cross-dataset ranking is claimed.
#
# The only arithmetic performed here is REPORTING-DERIVED arithmetic on
# already-frozen aggregate MAEs:
#
#     selective_mae_gain = full_mae - accepted_mae
#     relative_mae_reduction_pct = 100 * gain / full_mae
#
# This is not scientific re-scoring.

import os
import json
import math
import hashlib
from pathlib import Path

import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631i = (
    "all_checks_passed_631h",

    "FD003_TARGET_CONSTRUCTION_RECOVERY_631H",
    "FD003_TARGET_CONSTRUCTION_RECOVERY_SHA256_631H",
    "FD003_RESOLVED_TARGET_CLASS_631H",

    "FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G",
    "FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G",

    "FD001_FINAL_REPORTING_AUTHORITY_631D",
    "FD002_CAPPED_RUL_FINAL_AUTHORITY_631E",

    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
    "FD003_FINAL_DATASET_CLOSURE_CONTRACT_629BC",

    "FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11",
    "FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11",
    "FD004_FINAL_DATASET_CLOSURE_SHA256_630W11",
)

missing_631i = [
    name
    for name in required_631i
    if name not in globals()
]

if missing_631i:
    raise RuntimeError(
        "Missing STEP 6.31I authority object(s): "
        + ", ".join(missing_631i)
    )

if not bool(all_checks_passed_631h):
    raise RuntimeError(
        "STEP 6.31H must remain passed."
    )


def _json_sha256_631i(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


# =============================================================================
# 1) VERIFY TARGET-PROVENANCE AUTHORITIES
# =============================================================================

if (
    _json_sha256_631i(
        FD003_TARGET_CONSTRUCTION_RECOVERY_631H
    )
    !=
    str(
        FD003_TARGET_CONSTRUCTION_RECOVERY_SHA256_631H
    )
):
    raise RuntimeError(
        "FD003 target-construction recovery changed."
    )


if (
    _json_sha256_631i(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G
    )
    !=
    str(
        FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_SHA256_631G
    )
):
    raise RuntimeError(
        "Frozen cross-dataset target provenance changed."
    )


if (
    FD003_RESOLVED_TARGET_CLASS_631H
    !=
    "RAW_OR_UNCAPPED_RUL_EXPLICIT"
):
    raise RuntimeError(
        "FD003 target provenance is not uniquely resolved as raw/uncapped."
    )


fd004_target_class_631i = (
    FD004_EXACT_CROSS_DATASET_TARGET_PROVENANCE_631G[
        "FD004"
    ][
        "resolved_target_class"
    ]
)

if (
    fd004_target_class_631i
    !=
    "RAW_OR_UNCAPPED_RUL_EXPLICIT"
):
    raise RuntimeError(
        "FD004 target provenance is not frozen as raw/uncapped."
    )


fd002_target_631i = str(
    FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
        "evaluation_target"
    ]
)

if "capped" not in fd002_target_631i.lower():
    raise RuntimeError(
        "FD002 final authority is no longer explicitly capped."
    )


# =============================================================================
# 2) VERIFY FD004 FINAL CLOSURE HASH
# =============================================================================

if (
    _json_sha256_631i(
        FD004_FINAL_DATASET_CLOSURE_CONTRACT_630W11
    )
    !=
    str(
        FD004_FINAL_DATASET_CLOSURE_SHA256_630W11
    )
):
    raise RuntimeError(
        "FD004 final dataset closure changed."
    )


# =============================================================================
# 3) LOAD + VERIFY FD001 FINAL EVIDENCE FILE
# =============================================================================

fd001_path_631i = Path(
    FD001_FINAL_REPORTING_AUTHORITY_631D[
        "path"
    ]
)

if not fd001_path_631i.is_file():
    raise RuntimeError(
        "FD001 final evidence authority file is missing."
    )


with open(
    fd001_path_631i,
    "rb",
) as handle_631i:
    fd001_file_bytes_631i = (
        handle_631i.read()
    )


fd001_file_sha_631i = hashlib.sha256(
    fd001_file_bytes_631i
).hexdigest()


if (
    fd001_file_sha_631i
    !=
    str(
        FD001_FINAL_REPORTING_AUTHORITY_631D[
            "file_sha256"
        ]
    )
):
    raise RuntimeError(
        "FD001 final evidence file hash changed."
    )


fd001_payload_631i = json.loads(
    fd001_file_bytes_631i.decode(
        "utf-8"
    )
)


# Exact frozen fields observed in the final FD001 evidence artifact.
required_fd001_keys_631i = (
    "primary_target",
    "secondary_target",
    "test_engines",
    "coverage",
    "primary_full_mae",
    "primary_accepted_mae",
    "primary_abstained_mae",
    "source_H5B",
)

missing_fd001_keys_631i = [
    key
    for key in required_fd001_keys_631i
    if key not in fd001_payload_631i
]

if missing_fd001_keys_631i:
    raise RuntimeError(
        "FD001 final evidence missing key(s): "
        + ", ".join(missing_fd001_keys_631i)
    )


fd001_h5b_631i = dict(
    fd001_payload_631i[
        "source_H5B"
    ]
)

required_fd001_raw_keys_631i = (
    "raw_full_mae",
    "raw_accepted_mae",
    "raw_abstained_mae",
)

missing_fd001_raw_keys_631i = [
    key
    for key in required_fd001_raw_keys_631i
    if key not in fd001_h5b_631i
]

if missing_fd001_raw_keys_631i:
    raise RuntimeError(
        "FD001 raw secondary evidence missing key(s): "
        + ", ".join(missing_fd001_raw_keys_631i)
    )


# =============================================================================
# 4) FD003 ROBUST FROZEN-METRIC EXTRACTION
#
# Extract only from the already-frozen HELD-OUT score authority.
# Final conclusion is fallback only if a field name differs.
# No predictions / labels are used.
# =============================================================================

def _walk_leaf_scalars_631i(
    value,
    path=(),
    output=None,
):
    if output is None:
        output = []

    if isinstance(
        value,
        dict,
    ):
        for key, child in value.items():
            _walk_leaf_scalars_631i(
                child,
                path + (str(key),),
                output,
            )

    elif isinstance(
        value,
        (list, tuple),
    ):
        for idx, child in enumerate(
            value
        ):
            _walk_leaf_scalars_631i(
                child,
                path + (f"[{idx}]",),
                output,
            )

    else:
        output.append(
            (
                ".".join(path),
                path[-1] if path else "",
                value,
            )
        )

    return output


def _extract_unique_numeric_631i(
    named_objects,
    aliases,
    label,
):
    aliases_lower = tuple(
        alias.lower()
        for alias in aliases
    )

    for object_name, obj in named_objects:

        rows = _walk_leaf_scalars_631i(
            obj
        )

        for alias_lower in aliases_lower:

            matches = []

            for path, leaf, value in rows:

                if leaf.lower() != alias_lower:
                    continue

                if isinstance(
                    value,
                    bool,
                ):
                    continue

                if not isinstance(
                    value,
                    (int, float),
                ):
                    continue

                matches.append(
                    (
                        path,
                        float(value),
                    )
                )

            if not matches:
                continue


            reference = matches[
                0
            ][
                1
            ]

            if not all(
                math.isclose(
                    value,
                    reference,
                    rel_tol=0.0,
                    abs_tol=1e-12,
                )
                for _, value
                in matches
            ):
                raise RuntimeError(
                    f"Ambiguous frozen FD003 field for {label}: "
                    f"{matches}"
                )

            return (
                reference,
                object_name,
                tuple(
                    matches
                ),
            )


    raise RuntimeError(
        "Unable to recover frozen FD003 aggregate field: "
        + label
    )


fd003_metric_sources_631i = (
    (
        "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
        FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,
    ),
    (
        "FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC",
        FD003_FINAL_SCIENTIFIC_CONCLUSION_629BC,
    ),
)


(
    fd003_endpoint_count_631i,
    fd003_endpoint_source_631i,
    fd003_endpoint_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "test_engine_count",
        "test_endpoints",
        "endpoint_count",
        "test_endpoint_count",
        "expected_test_engine_count",
    ),
    "TEST endpoint count",
)


(
    fd003_accepted_count_631i,
    fd003_accepted_count_source_631i,
    fd003_accepted_count_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "accepted_count",
        "accepted_test_endpoints",
        "accepted_endpoint_count",
    ),
    "accepted count",
)


(
    fd003_abstained_count_631i,
    fd003_abstained_count_source_631i,
    fd003_abstained_count_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "abstained_count",
        "abstained_test_endpoints",
        "abstained_endpoint_count",
    ),
    "abstained count",
)


(
    fd003_coverage_631i,
    fd003_coverage_source_631i,
    fd003_coverage_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "realized_coverage",
        "realized_test_coverage",
        "coverage",
    ),
    "coverage",
)


(
    fd003_full_mae_631i,
    fd003_full_mae_source_631i,
    fd003_full_mae_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "full_mae",
        "full_test_mae",
        "mae_full",
    ),
    "full MAE",
)


(
    fd003_accepted_mae_631i,
    fd003_accepted_mae_source_631i,
    fd003_accepted_mae_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "accepted_mae",
        "accepted_test_mae",
        "mae_accepted",
    ),
    "accepted MAE",
)


(
    fd003_abstained_mae_631i,
    fd003_abstained_mae_source_631i,
    fd003_abstained_mae_matches_631i,
) = _extract_unique_numeric_631i(
    fd003_metric_sources_631i,
    (
        "abstained_mae",
        "abstained_test_mae",
        "mae_abstained",
    ),
    "abstained MAE",
)


fd003_endpoint_count_631i = int(
    fd003_endpoint_count_631i
)

fd003_accepted_count_631i = int(
    fd003_accepted_count_631i
)

fd003_abstained_count_631i = int(
    fd003_abstained_count_631i
)


if (
    fd003_accepted_count_631i
    +
    fd003_abstained_count_631i
    !=
    fd003_endpoint_count_631i
):
    raise RuntimeError(
        "FD003 frozen accepted/abstained counts do not sum to endpoints."
    )


# =============================================================================
# 5) REPORTING-DERIVED ARITHMETIC HELPER
# =============================================================================

def _reporting_gain_631i(
    full_mae,
    accepted_mae,
):
    if (
        full_mae is None
        or
        accepted_mae is None
    ):
        return (
            None,
            None,
        )

    full_mae = float(
        full_mae
    )

    accepted_mae = float(
        accepted_mae
    )

    gain = (
        full_mae
        -
        accepted_mae
    )

    relative = (
        100.0
        *
        gain
        /
        full_mae
    )

    return (
        float(
            gain
        ),
        float(
            relative
        ),
    )


# =============================================================================
# 6) BUILD TARGET-AWARE HELD-OUT RECORDS
# =============================================================================

# -------------------------------------------------------------------------
# FD001 capped primary
# -------------------------------------------------------------------------

fd001_capped_full_631i = float(
    fd001_payload_631i[
        "primary_full_mae"
    ]
)

fd001_capped_accepted_631i = float(
    fd001_payload_631i[
        "primary_accepted_mae"
    ]
)

fd001_capped_abstained_631i = float(
    fd001_payload_631i[
        "primary_abstained_mae"
    ]
)

(
    fd001_capped_gain_631i,
    fd001_capped_relative_631i,
) = _reporting_gain_631i(
    fd001_capped_full_631i,
    fd001_capped_accepted_631i,
)


# -------------------------------------------------------------------------
# FD001 raw secondary
# -------------------------------------------------------------------------

fd001_raw_full_631i = float(
    fd001_h5b_631i[
        "raw_full_mae"
    ]
)

fd001_raw_accepted_631i = float(
    fd001_h5b_631i[
        "raw_accepted_mae"
    ]
)

fd001_raw_abstained_631i = float(
    fd001_h5b_631i[
        "raw_abstained_mae"
    ]
)

(
    fd001_raw_gain_631i,
    fd001_raw_relative_631i,
) = _reporting_gain_631i(
    fd001_raw_full_631i,
    fd001_raw_accepted_631i,
)


# -------------------------------------------------------------------------
# FD002 capped held-out
# -------------------------------------------------------------------------

fd002_full_631i = float(
    FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
        "full_test_mae"
    ]
)

fd002_accepted_631i = float(
    FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
        "accepted_test_mae"
    ]
)

fd002_abstained_631i = float(
    FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
        "abstained_test_mae"
    ]
)

(
    fd002_gain_631i,
    fd002_relative_631i,
) = _reporting_gain_631i(
    fd002_full_631i,
    fd002_accepted_631i,
)


# -------------------------------------------------------------------------
# FD003 raw/uncapped held-out
# -------------------------------------------------------------------------

(
    fd003_gain_631i,
    fd003_relative_631i,
) = _reporting_gain_631i(
    fd003_full_mae_631i,
    fd003_accepted_mae_631i,
)


# -------------------------------------------------------------------------
# FD004 held-out TEST: deliberately no MAE because it was never scored.
# -------------------------------------------------------------------------

fd004_heldout_631i = dict(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "heldout_test"
    ]
)


heldout_records_631i = [
    {
        "dataset":
            "FD001",

        "evaluation":
            "held-out TEST",

        "target_family":
            "CAPPED_RUL",

        "target_definition":
            str(
                fd001_payload_631i[
                    "primary_target"
                ]
            ),

        "role":
            "primary",

        "endpoint_count":
            int(
                fd001_payload_631i[
                    "test_engines"
                ]
            ),

        "accepted_count":
            None,

        "abstained_count":
            None,

        "coverage":
            float(
                fd001_payload_631i[
                    "coverage"
                ]
            ),

        "full_mae":
            fd001_capped_full_631i,

        "accepted_mae":
            fd001_capped_accepted_631i,

        "abstained_mae":
            fd001_capped_abstained_631i,

        "selective_mae_gain":
            fd001_capped_gain_631i,

        "relative_mae_reduction_pct":
            fd001_capped_relative_631i,

        "heldout_status":
            "SCORED",

        "scientific_interpretation":
            "positive selective MAE gain",

        "metric_origin":
            "frozen final aggregate evidence",

        "gain_origin":
            "reporting-derived from frozen aggregate MAEs",
    },

    {
        "dataset":
            "FD001",

        "evaluation":
            "held-out TEST",

        "target_family":
            "RAW_OR_UNCAPPED_RUL",

        "target_definition":
            str(
                fd001_payload_631i[
                    "secondary_target"
                ]
            ),

        "role":
            "secondary target sensitivity",

        "endpoint_count":
            int(
                fd001_payload_631i[
                    "test_engines"
                ]
            ),

        "accepted_count":
            None,

        "abstained_count":
            None,

        "coverage":
            float(
                fd001_payload_631i[
                    "coverage"
                ]
            ),

        "full_mae":
            fd001_raw_full_631i,

        "accepted_mae":
            fd001_raw_accepted_631i,

        "abstained_mae":
            fd001_raw_abstained_631i,

        "selective_mae_gain":
            fd001_raw_gain_631i,

        "relative_mae_reduction_pct":
            fd001_raw_relative_631i,

        "heldout_status":
            "SCORED",

        "scientific_interpretation":
            "positive selective MAE gain",

        "metric_origin":
            "frozen final aggregate evidence",

        "gain_origin":
            "reporting-derived from frozen aggregate MAEs",
    },

    {
        "dataset":
            "FD002",

        "evaluation":
            "held-out TEST",

        "target_family":
            "CAPPED_RUL",

        "target_definition":
            fd002_target_631i,

        "role":
            "final held-out authority",

        "endpoint_count":
            int(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "test_endpoints"
                ]
            ),

        "accepted_count":
            int(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "accepted_test_endpoints"
                ]
            ),

        "abstained_count":
            int(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "abstained_test_endpoints"
                ]
            ),

        "coverage":
            float(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "realized_test_coverage"
                ]
            ),

        "full_mae":
            fd002_full_631i,

        "accepted_mae":
            fd002_accepted_631i,

        "abstained_mae":
            fd002_abstained_631i,

        "selective_mae_gain":
            fd002_gain_631i,

        "relative_mae_reduction_pct":
            fd002_relative_631i,

        "heldout_status":
            "SCORED",

        "scientific_interpretation":
            "positive selective MAE gain",

        "metric_origin":
            "frozen official TEST scoring authority",

        "gain_origin":
            "reporting-derived from frozen aggregate MAEs",
    },

    {
        "dataset":
            "FD003",

        "evaluation":
            "held-out TEST",

        "target_family":
            "RAW_OR_UNCAPPED_RUL",

        "target_definition":
            (
                "standard uncapped RUL: "
                "engine terminal observed cycle - current cycle"
            ),

        "role":
            "final held-out authority",

        "endpoint_count":
            fd003_endpoint_count_631i,

        "accepted_count":
            fd003_accepted_count_631i,

        "abstained_count":
            fd003_abstained_count_631i,

        "coverage":
            float(
                fd003_coverage_631i
            ),

        "full_mae":
            float(
                fd003_full_mae_631i
            ),

        "accepted_mae":
            float(
                fd003_accepted_mae_631i
            ),

        "abstained_mae":
            float(
                fd003_abstained_mae_631i
            ),

        "selective_mae_gain":
            fd003_gain_631i,

        "relative_mae_reduction_pct":
            fd003_relative_631i,

        "heldout_status":
            "SCORED",

        "scientific_interpretation":
            (
                "negative selective MAE gain; frozen reliability policy "
                "did not improve held-out TEST MAE"
            ),

        "metric_origin":
            "frozen held-out TEST score contract",

        "gain_origin":
            "reporting-derived from frozen aggregate MAEs",
    },

    {
        "dataset":
            "FD004",

        "evaluation":
            "held-out TEST",

        "target_family":
            "RAW_OR_UNCAPPED_RUL",

        "target_definition":
            (
                "uncapped engine_terminal_cycle - current_cycle"
            ),

        "role":
            "primary held-out protocol",

        "endpoint_count":
            int(
                fd004_heldout_631i[
                    "test_engine_count"
                ]
            ),

        "accepted_count":
            None,

        "abstained_count":
            None,

        "coverage":
            None,

        "full_mae":
            None,

        "accepted_mae":
            None,

        "abstained_mae":
            None,

        "selective_mae_gain":
            None,

        "relative_mae_reduction_pct":
            None,

        "heldout_status":
            "NOT_SCORED_STRUCTURAL_PROTOCOL_GUARD",

        "scientific_interpretation":
            (
                "primary held-out TEST unscored because 11 TEST engines "
                "had fewer than the frozen 30 cycles required for one "
                "terminal input window; inference stopped before scoring"
            ),

        "metric_origin":
            "frozen FD004 final closure",

        "gain_origin":
            None,
    },
]


heldout_df_631i = pd.DataFrame(
    heldout_records_631i
)


# =============================================================================
# 7) VALIDATE HELD-OUT TABLE SEMANTICS
# =============================================================================

if len(
    heldout_df_631i
) != 5:
    raise RuntimeError(
        "Expected exactly five target-aware held-out rows."
    )


fd004_row_631i = (
    heldout_df_631i[
        heldout_df_631i[
            "dataset"
        ]
        ==
        "FD004"
    ]
)

if len(
    fd004_row_631i
) != 1:
    raise RuntimeError(
        "FD004 held-out synthesis row is not unique."
    )


for col_631i in (
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_mae_gain",
    "relative_mae_reduction_pct",
):

    if not pd.isna(
        fd004_row_631i.iloc[
            0
        ][
            col_631i
        ]
    ):
        raise RuntimeError(
            "FD004 held-out synthesis unexpectedly contains a TEST metric: "
            + col_631i
        )


# Scientific direction checks from frozen aggregate results.
fd001_capped_positive_631i = bool(
    fd001_capped_gain_631i
    >
    0
)

fd001_raw_positive_631i = bool(
    fd001_raw_gain_631i
    >
    0
)

fd002_positive_631i = bool(
    fd002_gain_631i
    >
    0
)

fd003_positive_631i = bool(
    fd003_gain_631i
    >
    0
)


if not (
    fd001_capped_positive_631i
    and
    fd001_raw_positive_631i
    and
    fd002_positive_631i
):
    raise RuntimeError(
        "Historical positive held-out selective-result direction changed."
    )


if fd003_positive_631i:
    raise RuntimeError(
        "FD003 unexpectedly changed from its frozen non-improving result."
    )


# =============================================================================
# 8) FD004 DEV + CONTROLLED-CORRUPTION EVIDENCE
#
# Kept separate because it is NOT held-out TEST evidence.
# =============================================================================

fd004_clean_dev_631i = dict(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "clean_dev"
    ]
)

fd004_corruption_631i = dict(
    FD004_FINAL_SCIENTIFIC_CONCLUSION_630W11[
        "controlled_corruption"
    ]
)


fd004_supporting_evidence_631i = {
    "evaluation_stage":
        "DEV + controlled corruption",

    "target_family":
        "RAW_OR_UNCAPPED_RUL",

    "clean_dev": {
        "coverage":
            float(
                fd004_clean_dev_631i[
                    "realized_coverage"
                ]
            ),

        "full_mae":
            float(
                fd004_clean_dev_631i[
                    "full_mae"
                ]
            ),

        "accepted_mae":
            float(
                fd004_clean_dev_631i[
                    "accepted_mae"
                ]
            ),

        "abstained_mae":
            float(
                fd004_clean_dev_631i[
                    "abstained_mae"
                ]
            ),

        "selective_mae_gain":
            float(
                fd004_clean_dev_631i[
                    "selective_mae_gain"
                ]
            ),

        "relative_mae_reduction_pct":
            float(
                fd004_clean_dev_631i[
                    "relative_mae_reduction_percent"
                ]
            ),
    },

    "controlled_corruption": {
        "scenario_count":
            int(
                fd004_corruption_631i[
                    "scenario_count"
                ]
            ),

        "positive_gain_count":
            int(
                fd004_corruption_631i[
                    "positive_gain_count"
                ]
            ),

        "zero_gain_count":
            int(
                fd004_corruption_631i[
                    "zero_gain_count"
                ]
            ),

        "negative_gain_count":
            int(
                fd004_corruption_631i[
                    "negative_gain_count"
                ]
            ),

        "positive_gain_rate":
            float(
                fd004_corruption_631i[
                    "positive_gain_rate"
                ]
            ),

        "negative_gain_family":
            str(
                fd004_corruption_631i[
                    "negative_gain_family"
                ]
            ),

        "negative_gain_by_severity":
            dict(
                fd004_corruption_631i[
                    "negative_gain_by_severity"
                ]
            ),
    },

    "heldout_test_status":
        str(
            fd004_heldout_631i[
                "primary_result"
            ]
        ),
}


# =============================================================================
# 9) COMPARABILITY / INTERPRETATION CONTRACT
# =============================================================================

FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I = {
    "heldout_findings": {
        "FD001":
            (
                "Selective filtering improved held-out TEST MAE under both "
                "the primary capped target and the retained raw-RUL "
                "secondary analysis, at relatively low realized coverage."
            ),

        "FD002":
            (
                "The frozen reliability policy improved held-out TEST MAE "
                "for the piecewise-capped RUL evaluation."
            ),

        "FD003":
            (
                "The frozen reliability policy did not improve held-out "
                "TEST MAE under the uncapped/raw RUL target; accepted MAE "
                "was slightly worse than full-population MAE."
            ),

        "FD004":
            (
                "No held-out TEST performance estimate is reported under "
                "the frozen protocol because short TEST trajectories made "
                "the complete endpoint population structurally inadmissible "
                "for the fixed 30-cycle input rule."
            ),
    },

    "fd004_supporting_findings":
        (
            "Before TEST access, FD004 CLEAN DEV showed positive selective "
            "MAE gain. Across 381 executable controlled-degradation "
            "scenarios, 365 showed positive selective gain; all 16 negative "
            "cases were in the stuck-sensor family, concentrated most "
            "strongly at high severity."
        ),

    "cross_dataset_conclusion":
        (
            "Reliability-aware selective prediction showed useful error "
            "separation in FD001 and FD002 and strong DEV/corruption evidence "
            "in FD004, but it did not generalize uniformly: FD003 produced a "
            "scored held-out negative result, and FD004 exposed a structural "
            "evaluation limitation of the fixed-window protocol."
        ),

    "comparison_guard":
        (
            "Do not rank datasets by absolute MAE. C-MAPSS subsets differ in "
            "operating conditions, fault structure, target conventions, and "
            "evaluation circumstances. Absolute MAEs are reported as "
            "dataset-specific outcomes, not as a pooled performance ranking."
        ),

    "target_guard":
        (
            "Capped-target rows and raw/uncapped-target rows must remain "
            "explicitly distinguished. FD002 has no recovered raw-RUL final "
            "held-out authority, and FD004 has no scored held-out TEST MAE."
        ),

    "evaluation_stage_guard":
        (
            "FD004 CLEAN DEV and corruption results are supporting evidence "
            "only and must not be presented as held-out TEST performance."
        ),
}


# =============================================================================
# 10) SOURCE / PROVENANCE CONTRACT
# =============================================================================

FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I = {
    "stage":
        "6.31I",

    "status":
        "FINAL_TARGET_AWARE_CROSS_DATASET_SYNTHESIS_LOCKED",

    "FD001": {
        "authority_path":
            str(
                fd001_path_631i
            ),

        "authority_file_sha256":
            fd001_file_sha_631i,

        "target_status":
            "capped primary + raw secondary",
    },

    "FD002": {
        "authority_path":
            str(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "path"
                ]
            ),

        "authority_file_sha256":
            str(
                FD002_CAPPED_RUL_FINAL_AUTHORITY_631E[
                    "file_sha256"
                ]
            ),

        "target_status":
            "piecewise-capped held-out authority only",

        "raw_final_authority_available":
            False,
    },

    "FD003": {
        "target_recovery_sha256":
            str(
                FD003_TARGET_CONSTRUCTION_RECOVERY_SHA256_631H
            ),

        "target_status":
            FD003_RESOLVED_TARGET_CLASS_631H,

        "heldout_metric_primary_source":
            "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",

        "endpoint_source":
            fd003_endpoint_source_631i,

        "coverage_source":
            fd003_coverage_source_631i,

        "full_mae_source":
            fd003_full_mae_source_631i,

        "accepted_mae_source":
            fd003_accepted_mae_source_631i,

        "abstained_mae_source":
            fd003_abstained_mae_source_631i,
    },

    "FD004": {
        "final_closure_sha256":
            str(
                FD004_FINAL_DATASET_CLOSURE_SHA256_630W11
            ),

        "target_status":
            fd004_target_class_631i,

        "heldout_scored":
            False,

        "supporting_dev_corruption_evidence_included":
            True,
    },

    "reporting_derived_arithmetic": {
        "used":
            True,

        "formula_selective_mae_gain":
            "full_mae - accepted_mae",

        "formula_relative_mae_reduction_pct":
            "100 * (full_mae - accepted_mae) / full_mae",

        "input_level":
            "frozen aggregate MAEs only",

        "predictions_or_labels_used":
            False,
    },

    "scientific_recomputation":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "raw_test_rul_access":
        False,

    "threshold_policy_selection":
        False,

    "cross_dataset_absolute_mae_ranking_permitted":
        False,
}


FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I = (
    _json_sha256_631i(
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I
    )
)


# =============================================================================
# 11) SERIALIZABLE FINAL SYNTHESIS CONTRACT
# =============================================================================

heldout_records_json_631i = json.loads(
    heldout_df_631i.to_json(
        orient="records"
    )
)


FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I = {
    "stage":
        "6.31I",

    "status":
        "FINAL_TARGET_AWARE_SYNTHESIS_COMPLETE_REPORTING_ONLY",

    "heldout_target_aware_results":
        heldout_records_json_631i,

    "fd004_supporting_evidence":
        fd004_supporting_evidence_631i,

    "interpretation":
        FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I,

    "provenance_sha256":
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I,
}


FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I = (
    _json_sha256_631i(
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I
    )
)


# =============================================================================
# 12) WRITE FINAL REPORTING ARTIFACTS
# =============================================================================

output_dir_631i = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\cross_dataset_synthesis"
)

output_dir_631i.mkdir(
    parents=True,
    exist_ok=True,
)


heldout_csv_path_631i = (
    output_dir_631i
    /
    "fd001_fd004_target_aware_heldout_results_631i.csv"
)

synthesis_json_path_631i = (
    output_dir_631i
    /
    "fd001_fd004_final_target_aware_synthesis_631i.json"
)


heldout_df_631i.to_csv(
    heldout_csv_path_631i,
    index=False,
)


with open(
    synthesis_json_path_631i,
    "w",
    encoding="utf-8",
) as handle_631i:

    json.dump(
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I,
        handle_631i,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


# =============================================================================
# 13) RESTART-SAFE CHECKPOINT
# =============================================================================

checkpoint_payload_631i = {
    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I":
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I,

    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I":
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I,

    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I":
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I,

    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I":
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I,
}


checkpoint_path_631i = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd001_fd004_final_target_aware_synthesis_631i_v1.json"
)

checkpoint_path_631i.parent.mkdir(
    parents=True,
    exist_ok=True,
)


tmp_checkpoint_path_631i = Path(
    str(
        checkpoint_path_631i
    )
    +
    ".tmp"
)


with open(
    tmp_checkpoint_path_631i,
    "w",
    encoding="utf-8",
) as handle_631i:

    json.dump(
        checkpoint_payload_631i,
        handle_631i,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


os.replace(
    tmp_checkpoint_path_631i,
    checkpoint_path_631i,
)


with open(
    checkpoint_path_631i,
    "r",
    encoding="utf-8",
) as handle_631i:

    checkpoint_roundtrip_631i = json.load(
        handle_631i
    )


checkpoint_roundtrip_exact_631i = bool(
    checkpoint_roundtrip_631i
    ==
    checkpoint_payload_631i
)

if not checkpoint_roundtrip_exact_631i:
    raise RuntimeError(
        "Final cross-dataset synthesis checkpoint roundtrip failed."
    )


with open(
    checkpoint_path_631i,
    "rb",
) as handle_631i:

    FD001_FD004_FINAL_SYNTHESIS_CHECKPOINT_FILE_SHA256_631I = (
        hashlib.sha256(
            handle_631i.read()
        ).hexdigest()
    )


# =============================================================================
# 14) OUTPUT ARTIFACT HASHES
# =============================================================================

with open(
    heldout_csv_path_631i,
    "rb",
) as handle_631i:

    FD001_FD004_HELDOUT_CSV_SHA256_631I = (
        hashlib.sha256(
            handle_631i.read()
        ).hexdigest()
    )


with open(
    synthesis_json_path_631i,
    "rb",
) as handle_631i:

    FD001_FD004_SYNTHESIS_JSON_FILE_SHA256_631I = (
        hashlib.sha256(
            handle_631i.read()
        ).hexdigest()
    )


# =============================================================================
# 15) FINAL LOCK FLAGS
# =============================================================================

FD001_FD004_FINAL_SYNTHESIS_LOCKED_631I = True
FD001_FD004_REPORTING_ONLY_AFTER_631I = True
FD001_FD004_READY_FOR_FINAL_WRITING_631I = True

all_checks_passed_631i = True


# =============================================================================
# 16) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31I — FINAL TARGET-AWARE "
    "FD001–FD004 SYNTHESIS LOCK"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Predictions regenerated           :", False)
print("Labels reconstructed              :", False)
print("Raw TEST/RUL access               :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Reporting-derived arithmetic only :", True)


print("\n--- TARGET PROVENANCE ---")

print(
    "FD001                             :",
    "capped primary + raw secondary",
)

print(
    "FD002                             :",
    "piecewise-capped held-out authority only",
)

print(
    "FD003                             :",
    FD003_RESOLVED_TARGET_CLASS_631H,
)

print(
    "FD004                             :",
    fd004_target_class_631i,
)


print("\n--- FINAL TARGET-AWARE HELD-OUT RESULTS ---")

display_columns_631i = [
    "dataset",
    "target_family",
    "role",
    "endpoint_count",
    "accepted_count",
    "abstained_count",
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "selective_mae_gain",
    "relative_mae_reduction_pct",
    "heldout_status",
]

print(
    heldout_df_631i[
        display_columns_631i
    ].to_string(
        index=False
    )
)


print("\n--- HELD-OUT SCIENTIFIC DIRECTION ---")

print(
    "FD001 capped selective gain > 0  :",
    fd001_capped_positive_631i,
)

print(
    "FD001 raw selective gain > 0     :",
    fd001_raw_positive_631i,
)

print(
    "FD002 capped selective gain > 0  :",
    fd002_positive_631i,
)

print(
    "FD003 raw selective gain > 0     :",
    fd003_positive_631i,
)

print(
    "FD004 held-out scored            :",
    False,
)


print("\n--- FD004 SUPPORTING DEV / CORRUPTION EVIDENCE ---")

print(
    "Clean DEV coverage                :",
    fd004_supporting_evidence_631i[
        "clean_dev"
    ][
        "coverage"
    ],
)

print(
    "Clean DEV full / accepted MAE     :",
    (
        fd004_supporting_evidence_631i[
            "clean_dev"
        ][
            "full_mae"
        ],
        fd004_supporting_evidence_631i[
            "clean_dev"
        ][
            "accepted_mae"
        ],
    ),
)

print(
    "Clean DEV selective gain          :",
    fd004_supporting_evidence_631i[
        "clean_dev"
    ][
        "selective_mae_gain"
    ],
)

print(
    "Corruption scenarios              :",
    fd004_supporting_evidence_631i[
        "controlled_corruption"
    ][
        "scenario_count"
    ],
)

print(
    "Positive / zero / negative        :",
    (
        fd004_supporting_evidence_631i[
            "controlled_corruption"
        ][
            "positive_gain_count"
        ],
        fd004_supporting_evidence_631i[
            "controlled_corruption"
        ][
            "zero_gain_count"
        ],
        fd004_supporting_evidence_631i[
            "controlled_corruption"
        ][
            "negative_gain_count"
        ],
    ),
)

print(
    "Negative-gain family              :",
    fd004_supporting_evidence_631i[
        "controlled_corruption"
    ][
        "negative_gain_family"
    ],
)


print("\n--- CROSS-DATASET INTERPRETATION ---")

print(
    FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I[
        "cross_dataset_conclusion"
    ]
)

print()

print(
    "COMPARISON GUARD:"
)

print(
    FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I[
        "comparison_guard"
    ]
)

print()

print(
    "TARGET GUARD:"
)

print(
    FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I[
        "target_guard"
    ]
)


print("\n--- FINAL REPORTING ARTIFACTS ---")

print(
    "Held-out CSV                      :",
    heldout_csv_path_631i,
)

print(
    "Held-out CSV SHA-256              :",
    FD001_FD004_HELDOUT_CSV_SHA256_631I,
)

print(
    "Synthesis JSON                    :",
    synthesis_json_path_631i,
)

print(
    "Synthesis JSON file SHA-256       :",
    FD001_FD004_SYNTHESIS_JSON_FILE_SHA256_631I,
)

print(
    "Synthesis contract SHA-256        :",
    FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I,
)

print(
    "Provenance contract SHA-256       :",
    FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I,
)

print(
    "Checkpoint                        :",
    checkpoint_path_631i,
)

print(
    "Checkpoint SHA-256                :",
    FD001_FD004_FINAL_SYNTHESIS_CHECKPOINT_FILE_SHA256_631I,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_631i,
)


print("\n--- FINAL STATE ---")

print(
    "Cross-dataset synthesis locked    :",
    FD001_FD004_FINAL_SYNTHESIS_LOCKED_631I,
)

print(
    "Reporting-only from here          :",
    FD001_FD004_REPORTING_ONLY_AFTER_631I,
)

print(
    "Ready for final research writing  :",
    FD001_FD004_READY_FOR_FINAL_WRITING_631I,
)


print(
    "\nPASS — FD001–FD004 target-aware scientific synthesis is frozen."
)

print(
    "The empirical phase is complete. Subsequent work is final "
    "results/discussion/limitations/table/figure writing only."
)

In [ ]:
# STEP 6.31J — MANUSCRIPT-READY RESULTS / DISCUSSION / LIMITATIONS BUNDLE
#
# Purpose:
# Convert the already-LOCKED 6.31I cross-dataset synthesis into
# manuscript-ready reporting artifacts.
#
# This step creates:
#   1) target-aware held-out results table
#   2) FD004 supporting DEV/corruption table
#   3) results/discussion/limitations reporting bundle
#   4) Markdown manuscript-ready summary
#   5) restart-safe reporting checkpoint
#
# REPORTING ONLY.
#
# NO predictions.
# NO labels.
# NO raw TEST/RUL access.
# NO inference.
# NO MC dropout.
# NO metric recomputation.
# NO threshold/policy/model change.
# NO cross-dataset ranking.

import os
import json
import math
import hashlib
from pathlib import Path

import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631j = (
    "all_checks_passed_631i",

    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I",
    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I",

    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I",
    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I",

    "FD001_FD004_FINAL_SYNTHESIS_INTERPRETATION_631I",

    "FD001_FD004_FINAL_SYNTHESIS_LOCKED_631I",
    "FD001_FD004_REPORTING_ONLY_AFTER_631I",
    "FD001_FD004_READY_FOR_FINAL_WRITING_631I",
)

missing_631j = [
    name
    for name in required_631j
    if name not in globals()
]

if missing_631j:
    raise RuntimeError(
        "Missing STEP 6.31J authority object(s): "
        + ", ".join(missing_631j)
    )


if not bool(all_checks_passed_631i):
    raise RuntimeError(
        "STEP 6.31I must remain passed."
    )


if not bool(
    FD001_FD004_FINAL_SYNTHESIS_LOCKED_631I
):
    raise RuntimeError(
        "6.31I cross-dataset synthesis is not locked."
    )


if not bool(
    FD001_FD004_REPORTING_ONLY_AFTER_631I
):
    raise RuntimeError(
        "Project is not in reporting-only state."
    )


if not bool(
    FD001_FD004_READY_FOR_FINAL_WRITING_631I
):
    raise RuntimeError(
        "6.31I did not authorize final writing."
    )


# =============================================================================
# 1) HASH VERIFICATION
# =============================================================================

def _json_sha256_631j(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631j(
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I
    )
    !=
    str(
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I
    )
):
    raise RuntimeError(
        "Frozen 6.31I synthesis changed."
    )


if (
    _json_sha256_631j(
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_631I
    )
    !=
    str(
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I
    )
):
    raise RuntimeError(
        "Frozen 6.31I provenance changed."
    )


# =============================================================================
# 2) BIND FROZEN REPORTING CONTENT
# =============================================================================

synthesis_631j = (
    FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_631I
)

heldout_records_631j = tuple(
    synthesis_631j[
        "heldout_target_aware_results"
    ]
)

fd004_supporting_631j = dict(
    synthesis_631j[
        "fd004_supporting_evidence"
    ]
)

interpretation_631j = dict(
    synthesis_631j[
        "interpretation"
    ]
)


if len(
    heldout_records_631j
) != 5:
    raise RuntimeError(
        "Frozen target-aware held-out table no longer has five rows."
    )


# =============================================================================
# 3) MANUSCRIPT TABLE 1 — TARGET-AWARE HELD-OUT RESULTS
#
# No metric arithmetic.
# Values come directly from frozen 6.31I records.
# =============================================================================

def _target_label_631j(
    target_family,
):
    mapping = {
        "CAPPED_RUL":
            "Capped RUL",

        "RAW_OR_UNCAPPED_RUL":
            "Raw / uncapped RUL",
    }

    return mapping.get(
        str(
            target_family
        ),
        str(
            target_family
        ),
    )


def _display_status_631j(
    status,
):
    if status == "SCORED":
        return "Scored"

    if (
        status
        ==
        "NOT_SCORED_STRUCTURAL_PROTOCOL_GUARD"
    ):
        return "Not scored — structural protocol guard"

    return str(
        status
    )


table1_rows_631j = []

for row_631j in heldout_records_631j:

    table1_rows_631j.append(
        {
            "Dataset":
                row_631j[
                    "dataset"
                ],

            "Target":
                _target_label_631j(
                    row_631j[
                        "target_family"
                    ]
                ),

            "Analysis role":
                row_631j[
                    "role"
                ],

            "TEST endpoints":
                row_631j[
                    "endpoint_count"
                ],

            "Accepted":
                row_631j[
                    "accepted_count"
                ],

            "Abstained":
                row_631j[
                    "abstained_count"
                ],

            "Coverage":
                row_631j[
                    "coverage"
                ],

            "Full MAE":
                row_631j[
                    "full_mae"
                ],

            "Accepted MAE":
                row_631j[
                    "accepted_mae"
                ],

            "Abstained MAE":
                row_631j[
                    "abstained_mae"
                ],

            "Selective MAE gain":
                row_631j[
                    "selective_mae_gain"
                ],

            "Relative MAE reduction (%)":
                row_631j[
                    "relative_mae_reduction_pct"
                ],

            "Held-out status":
                _display_status_631j(
                    row_631j[
                        "heldout_status"
                    ]
                ),

            "Interpretation":
                row_631j[
                    "scientific_interpretation"
                ],
        }
    )


FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J = (
    pd.DataFrame(
        table1_rows_631j
    )
)


# =============================================================================
# 4) MANUSCRIPT TABLE 2 — FD004 SUPPORTING EVIDENCE
# =============================================================================

fd004_clean_631j = dict(
    fd004_supporting_631j[
        "clean_dev"
    ]
)

fd004_corr_631j = dict(
    fd004_supporting_631j[
        "controlled_corruption"
    ]
)


table2_rows_631j = [
    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Realized coverage",

        "Value":
            fd004_clean_631j[
                "coverage"
            ],
    },

    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Full MAE",

        "Value":
            fd004_clean_631j[
                "full_mae"
            ],
    },

    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Accepted MAE",

        "Value":
            fd004_clean_631j[
                "accepted_mae"
            ],
    },

    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Abstained MAE",

        "Value":
            fd004_clean_631j[
                "abstained_mae"
            ],
    },

    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Selective MAE gain",

        "Value":
            fd004_clean_631j[
                "selective_mae_gain"
            ],
    },

    {
        "Evidence block":
            "Clean DEV",

        "Measure":
            "Relative MAE reduction (%)",

        "Value":
            fd004_clean_631j[
                "relative_mae_reduction_pct"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Executable scenarios",

        "Value":
            fd004_corr_631j[
                "scenario_count"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Positive-gain scenarios",

        "Value":
            fd004_corr_631j[
                "positive_gain_count"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Zero-gain scenarios",

        "Value":
            fd004_corr_631j[
                "zero_gain_count"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Negative-gain scenarios",

        "Value":
            fd004_corr_631j[
                "negative_gain_count"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Positive-gain rate",

        "Value":
            fd004_corr_631j[
                "positive_gain_rate"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Negative-gain family",

        "Value":
            fd004_corr_631j[
                "negative_gain_family"
            ],
    },

    {
        "Evidence block":
            "Controlled corruption",

        "Measure":
            "Negative gain by severity",

        "Value":
            json.dumps(
                fd004_corr_631j[
                    "negative_gain_by_severity"
                ],
                sort_keys=True,
            ),
    },

    {
        "Evidence block":
            "Held-out TEST",

        "Measure":
            "Status",

        "Value":
            fd004_supporting_631j[
                "heldout_test_status"
            ],
    },
]


FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J = (
    pd.DataFrame(
        table2_rows_631j
    )
)


# =============================================================================
# 5) MANUSCRIPT CLAIMS / DISCUSSION / LIMITATIONS
#
# These statements remain bounded by the frozen 6.31I interpretation.
# =============================================================================

FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J = {
    "project_state":
        "EMPIRICAL_PHASE_COMPLETE_REPORTING_ONLY",

    "results": {
        "cross_dataset_summary":
            interpretation_631j[
                "cross_dataset_conclusion"
            ],

        "FD001":
            interpretation_631j[
                "heldout_findings"
            ][
                "FD001"
            ],

        "FD002":
            interpretation_631j[
                "heldout_findings"
            ][
                "FD002"
            ],

        "FD003":
            interpretation_631j[
                "heldout_findings"
            ][
                "FD003"
            ],

        "FD004":
            interpretation_631j[
                "heldout_findings"
            ][
                "FD004"
            ],

        "FD004_supporting_evidence":
            interpretation_631j[
                "fd004_supporting_findings"
            ],
    },

    "discussion": {
        "main_interpretation":
            (
                "The evidence supports reliability-aware selective RUL "
                "prediction as a useful error-separation mechanism in some "
                "settings, but not as a universally improving policy. "
                "FD001 and FD002 produced positive held-out selective gains, "
                "whereas FD003 produced a genuine held-out negative result."
            ),

        "degradation_interpretation":
            (
                "FD004 strengthens the reliability argument at DEV stage "
                "and under controlled sensor degradation: most executable "
                "corruption scenarios retained positive selective gain. "
                "However, stuck-sensor corruption exposed the principal "
                "observed degradation failure mode."
            ),

        "protocol_interpretation":
            (
                "FD004 also demonstrates that reliability evaluation depends "
                "on the admissibility of the underlying prediction protocol. "
                "The fixed 30-cycle input rule could not represent the full "
                "held-out endpoint population because 11 TEST trajectories "
                "were shorter than 30 cycles, so the frozen protocol required "
                "stopping before inference."
            ),
    },

    "limitations": {
        "target_heterogeneity":
            interpretation_631j[
                "target_guard"
            ],

        "absolute_mae_noncomparability":
            interpretation_631j[
                "comparison_guard"
            ],

        "evaluation_stage_separation":
            interpretation_631j[
                "evaluation_stage_guard"
            ],

        "FD002_raw_target_gap":
            (
                "No genuine pre-existing FD002 raw-RUL final held-out "
                "authority was recovered. FD002 therefore contributes only "
                "its frozen piecewise-capped held-out result."
            ),

        "FD003_negative_generalization":
            (
                "FD003 is an important negative generalization result: "
                "the frozen policy slightly worsened accepted-set held-out "
                "MAE relative to the full TEST population."
            ),

        "FD004_short_history_limitation":
            (
                "The current fixed-length model protocol has no prospectively "
                "frozen mechanism for TEST histories shorter than 30 cycles. "
                "Padding, engine exclusion, row repetition, or variable-length "
                "inference were not introduced after TEST access."
            ),

        "FD004_stuck_sensor_limitation":
            (
                "All 16 negative selective-gain controlled-corruption "
                "scenarios in FD004 occurred in the stuck-sensor family, "
                "with failures concentrated most strongly at high severity."
            ),
    },

    "nonclaims": (
        "Do not claim that the selective policy improves every C-MAPSS subset.",
        "Do not rank datasets by absolute MAE.",
        "Do not treat capped and raw/uncapped target results as interchangeable.",
        "Do not report an FD004 held-out TEST MAE.",
        "Do not present FD004 DEV or corruption outcomes as held-out TEST outcomes.",
        "Do not create a new FD002 raw-RUL score from stored endpoint predictions.",
        "Do not retrofit padding or variable-length inference into the frozen FD004 primary TEST result.",
    ),

    "recommended_table_notes": (
        (
            "FD001 reports both its primary capped-RUL evaluation and its "
            "retained raw-RUL secondary analysis."
        ),
        (
            "FD002 held-out results use the frozen piecewise-capped RUL target."
        ),
        (
            "FD003 uses uncapped TRAIN-derived RUL and produced a negative "
            "held-out selective-MAE result."
        ),
        (
            "FD004 held-out TEST was not scored because the frozen 30-cycle "
            "protocol was structurally incompatible with 11 of 248 TEST "
            "trajectories."
        ),
        (
            "FD004 DEV/corruption results are reported separately as supporting "
            "evidence and are not substitutes for held-out TEST performance."
        ),
    ),
}


# =============================================================================
# 6) MARKDOWN FORMATTERS
# =============================================================================

def _is_missing_631j(value):
    if value is None:
        return True

    try:
        return bool(
            pd.isna(
                value
            )
        )
    except Exception:
        return False


def _fmt_count_631j(value):
    if _is_missing_631j(
        value
    ):
        return "—"

    return str(
        int(
            float(
                value
            )
        )
    )


def _fmt_fraction_pct_631j(value):
    if _is_missing_631j(
        value
    ):
        return "—"

    return (
        f"{100.0 * float(value):.1f}%"
    )


def _fmt_metric_631j(value):
    if _is_missing_631j(
        value
    ):
        return "—"

    return f"{float(value):.3f}"


def _fmt_percent_631j(value):
    if _is_missing_631j(
        value
    ):
        return "—"

    return f"{float(value):.2f}%"


def _md_escape_631j(value):
    return str(
        value
    ).replace(
        "|",
        "\\|",
    ).replace(
        "\n",
        " ",
    )


def _markdown_table_631j(
    headers,
    rows,
):
    output = []

    output.append(
        "| "
        +
        " | ".join(
            _md_escape_631j(
                header
            )
            for header
            in headers
        )
        +
        " |"
    )

    output.append(
        "| "
        +
        " | ".join(
            "---"
            for _
            in headers
        )
        +
        " |"
    )

    for row in rows:

        output.append(
            "| "
            +
            " | ".join(
                _md_escape_631j(
                    value
                )
                for value
                in row
            )
            +
            " |"
        )

    return "\n".join(
        output
    )


# =============================================================================
# 7) MANUSCRIPT-READY MARKDOWN TABLE 1
# =============================================================================

table1_md_rows_631j = []

for row_631j in heldout_records_631j:

    table1_md_rows_631j.append(
        (
            row_631j[
                "dataset"
            ],

            _target_label_631j(
                row_631j[
                    "target_family"
                ]
            ),

            row_631j[
                "role"
            ],

            _fmt_count_631j(
                row_631j[
                    "endpoint_count"
                ]
            ),

            _fmt_fraction_pct_631j(
                row_631j[
                    "coverage"
                ]
            ),

            _fmt_metric_631j(
                row_631j[
                    "full_mae"
                ]
            ),

            _fmt_metric_631j(
                row_631j[
                    "accepted_mae"
                ]
            ),

            _fmt_metric_631j(
                row_631j[
                    "abstained_mae"
                ]
            ),

            _fmt_metric_631j(
                row_631j[
                    "selective_mae_gain"
                ]
            ),

            _fmt_percent_631j(
                row_631j[
                    "relative_mae_reduction_pct"
                ]
            ),

            _display_status_631j(
                row_631j[
                    "heldout_status"
                ]
            ),
        )
    )


table1_markdown_631j = _markdown_table_631j(
    (
        "Dataset",
        "Target",
        "Role",
        "Endpoints",
        "Coverage",
        "Full MAE",
        "Accepted MAE",
        "Abstained MAE",
        "Selective gain",
        "Relative reduction",
        "Status",
    ),
    table1_md_rows_631j,
)


# =============================================================================
# 8) MANUSCRIPT-READY MARKDOWN TABLE 2
# =============================================================================

table2_md_rows_631j = (
    (
        "Clean DEV",
        "Coverage",
        _fmt_fraction_pct_631j(
            fd004_clean_631j[
                "coverage"
            ]
        ),
    ),
    (
        "Clean DEV",
        "Full MAE",
        _fmt_metric_631j(
            fd004_clean_631j[
                "full_mae"
            ]
        ),
    ),
    (
        "Clean DEV",
        "Accepted MAE",
        _fmt_metric_631j(
            fd004_clean_631j[
                "accepted_mae"
            ]
        ),
    ),
    (
        "Clean DEV",
        "Abstained MAE",
        _fmt_metric_631j(
            fd004_clean_631j[
                "abstained_mae"
            ]
        ),
    ),
    (
        "Clean DEV",
        "Selective MAE gain",
        _fmt_metric_631j(
            fd004_clean_631j[
                "selective_mae_gain"
            ]
        ),
    ),
    (
        "Clean DEV",
        "Relative MAE reduction",
        _fmt_percent_631j(
            fd004_clean_631j[
                "relative_mae_reduction_pct"
            ]
        ),
    ),
    (
        "Controlled corruption",
        "Executable scenarios",
        str(
            fd004_corr_631j[
                "scenario_count"
            ]
        ),
    ),
    (
        "Controlled corruption",
        "Positive / zero / negative",
        (
            f"{fd004_corr_631j['positive_gain_count']} / "
            f"{fd004_corr_631j['zero_gain_count']} / "
            f"{fd004_corr_631j['negative_gain_count']}"
        ),
    ),
    (
        "Controlled corruption",
        "Positive-gain rate",
        _fmt_fraction_pct_631j(
            fd004_corr_631j[
                "positive_gain_rate"
            ]
        ),
    ),
    (
        "Controlled corruption",
        "Negative-gain family",
        str(
            fd004_corr_631j[
                "negative_gain_family"
            ]
        ),
    ),
    (
        "Controlled corruption",
        "Negative gain by severity",
        json.dumps(
            fd004_corr_631j[
                "negative_gain_by_severity"
            ],
            sort_keys=True,
        ),
    ),
    (
        "Held-out TEST",
        "Status",
        str(
            fd004_supporting_631j[
                "heldout_test_status"
            ]
        ),
    ),
)


table2_markdown_631j = _markdown_table_631j(
    (
        "Evidence block",
        "Measure",
        "Value",
    ),
    table2_md_rows_631j,
)


# =============================================================================
# 9) BUILD MARKDOWN REPORT
# =============================================================================

report_lines_631j = [
    "# FD001–FD004 Final Results Reporting Bundle",
    "",
    "## Table 1. Target-aware held-out results",
    "",
    table1_markdown_631j,
    "",
    (
        "**Table note.** Absolute MAE values are dataset-specific and "
        "must not be interpreted as a cross-dataset ranking. Capped and "
        "raw/uncapped target results are shown explicitly."
    ),
    "",
    "## Table 2. FD004 supporting DEV and controlled-degradation evidence",
    "",
    table2_markdown_631j,
    "",
    (
        "**Table note.** FD004 DEV and controlled-corruption outcomes are "
        "supporting evidence only and are not held-out TEST results."
    ),
    "",
    "## Results",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "cross_dataset_summary"
    ],
    "",
    "### FD001",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "FD001"
    ],
    "",
    "### FD002",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "FD002"
    ],
    "",
    "### FD003",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "FD003"
    ],
    "",
    "### FD004",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "FD004"
    ],
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "FD004_supporting_evidence"
    ],
    "",
    "## Discussion",
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "discussion"
    ][
        "main_interpretation"
    ],
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "discussion"
    ][
        "degradation_interpretation"
    ],
    "",
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "discussion"
    ][
        "protocol_interpretation"
    ],
    "",
    "## Limitations",
    "",
]

for (
    limitation_name_631j,
    limitation_text_631j,
) in FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
    "limitations"
].items():

    report_lines_631j.append(
        "- "
        +
        str(
            limitation_text_631j
        )
    )


report_lines_631j.extend(
    [
        "",
        "## Claims that must not be made",
        "",
    ]
)


for nonclaim_631j in (
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "nonclaims"
    ]
):

    report_lines_631j.append(
        "- "
        +
        str(
            nonclaim_631j
        )
    )


report_lines_631j.extend(
    [
        "",
        "## Recommended table notes",
        "",
    ]
)


for note_631j in (
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "recommended_table_notes"
    ]
):

    report_lines_631j.append(
        "- "
        +
        str(
            note_631j
        )
    )


manuscript_markdown_631j = "\n".join(
    report_lines_631j
)


# =============================================================================
# 10) FREEZE REPORTING CONTRACT
# =============================================================================

FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J = {
    "stage":
        "6.31J",

    "status":
        "MANUSCRIPT_READY_REPORTING_BUNDLE_COMPLETE",

    "source_synthesis_sha256":
        str(
            FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I
        ),

    "source_provenance_sha256":
        str(
            FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I
        ),

    "heldout_table_row_count":
        int(
            len(
                FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J
            )
        ),

    "fd004_supporting_table_row_count":
        int(
            len(
                FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J
            )
        ),

    "report_sections":
        (
            "target-aware held-out table",
            "FD004 supporting evidence table",
            "results",
            "discussion",
            "limitations",
            "nonclaims",
            "recommended table notes",
        ),

    "scientific_recomputation":
        False,

    "predictions_regenerated":
        False,

    "labels_reconstructed":
        False,

    "raw_test_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_selection":
        False,

    "cross_dataset_absolute_mae_ranking":
        False,

    "reporting_only":
        True,
}


FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J = (
    _json_sha256_631j(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J
    )
)


# =============================================================================
# 11) WRITE REPORTING ARTIFACTS
# =============================================================================

output_dir_631j = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\cross_dataset_synthesis"
    r"\manuscript_ready"
)

output_dir_631j.mkdir(
    parents=True,
    exist_ok=True,
)


heldout_table_path_631j = (
    output_dir_631j
    /
    "table1_target_aware_heldout_results_631j.csv"
)

fd004_table_path_631j = (
    output_dir_631j
    /
    "table2_fd004_supporting_evidence_631j.csv"
)

reporting_json_path_631j = (
    output_dir_631j
    /
    "fd001_fd004_manuscript_reporting_bundle_631j.json"
)

reporting_md_path_631j = (
    output_dir_631j
    /
    "fd001_fd004_manuscript_reporting_bundle_631j.md"
)


FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J.to_csv(
    heldout_table_path_631j,
    index=False,
)


FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J.to_csv(
    fd004_table_path_631j,
    index=False,
)


reporting_payload_631j = {
    "reporting_contract":
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J,

    "reporting_contract_sha256":
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J,

    "reporting_bundle":
        FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J,

    "table1_records":
        json.loads(
            FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J.to_json(
                orient="records"
            )
        ),

    "table2_records":
        json.loads(
            FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J.to_json(
                orient="records"
            )
        ),
}


with open(
    reporting_json_path_631j,
    "w",
    encoding="utf-8",
) as handle_631j:

    json.dump(
        reporting_payload_631j,
        handle_631j,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


with open(
    reporting_md_path_631j,
    "w",
    encoding="utf-8",
) as handle_631j:

    handle_631j.write(
        manuscript_markdown_631j
    )


# =============================================================================
# 12) RESTART-SAFE CHECKPOINT
# =============================================================================

checkpoint_payload_631j = {
    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J":
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J,

    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J":
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J,

    "FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J":
        FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J,
}


checkpoint_path_631j = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd001_fd004_manuscript_reporting_631j_v1.json"
)

checkpoint_path_631j.parent.mkdir(
    parents=True,
    exist_ok=True,
)


tmp_checkpoint_path_631j = Path(
    str(
        checkpoint_path_631j
    )
    +
    ".tmp"
)


with open(
    tmp_checkpoint_path_631j,
    "w",
    encoding="utf-8",
) as handle_631j:

    json.dump(
        checkpoint_payload_631j,
        handle_631j,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


os.replace(
    tmp_checkpoint_path_631j,
    checkpoint_path_631j,
)


with open(
    checkpoint_path_631j,
    "r",
    encoding="utf-8",
) as handle_631j:

    checkpoint_roundtrip_631j = json.load(
        handle_631j
    )


checkpoint_roundtrip_exact_631j = bool(
    checkpoint_roundtrip_631j
    ==
    checkpoint_payload_631j
)

if not checkpoint_roundtrip_exact_631j:
    raise RuntimeError(
        "6.31J reporting checkpoint roundtrip failed."
    )


# =============================================================================
# 13) FILE HASHES
# =============================================================================

def _sha_file_631j(path):
    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


FD001_FD004_MANUSCRIPT_TABLE1_FILE_SHA256_631J = (
    _sha_file_631j(
        heldout_table_path_631j
    )
)

FD004_MANUSCRIPT_TABLE2_FILE_SHA256_631J = (
    _sha_file_631j(
        fd004_table_path_631j
    )
)

FD001_FD004_MANUSCRIPT_JSON_FILE_SHA256_631J = (
    _sha_file_631j(
        reporting_json_path_631j
    )
)

FD001_FD004_MANUSCRIPT_MD_FILE_SHA256_631J = (
    _sha_file_631j(
        reporting_md_path_631j
    )
)

FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631J = (
    _sha_file_631j(
        checkpoint_path_631j
    )
)


# =============================================================================
# 14) FINAL WRITING FLAGS
# =============================================================================

FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J = True
FD001_FD004_READY_FOR_RESULTS_CHAPTER_631J = True
FD001_FD004_READY_FOR_DISCUSSION_CHAPTER_631J = True
FD001_FD004_READY_FOR_LIMITATIONS_SECTION_631J = True

all_checks_passed_631j = True


# =============================================================================
# 15) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31J — MANUSCRIPT-READY RESULTS / "
    "DISCUSSION / LIMITATIONS BUNDLE"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Predictions regenerated           :", False)
print("Labels reconstructed              :", False)
print("Raw TEST/RUL access               :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Cross-dataset MAE ranking         :", False)
print("Reporting-only                    :", True)


print("\n--- MANUSCRIPT TABLE 1 ---")

print(
    FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J.to_string(
        index=False
    )
)


print("\n--- MANUSCRIPT TABLE 2 ---")

print(
    FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J.to_string(
        index=False
    )
)


print("\n--- RESULTS SYNTHESIS ---")

print(
    FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
        "results"
    ][
        "cross_dataset_summary"
    ]
)


print("\n--- CORE LIMITATIONS ---")

for (
    limitation_name_631j,
    limitation_text_631j,
) in FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J[
    "limitations"
].items():

    print(
        f"{limitation_name_631j}: "
        f"{limitation_text_631j}"
    )


print("\n--- ARTIFACTS ---")

print(
    "Table 1 CSV                       :",
    heldout_table_path_631j,
)

print(
    "Table 1 SHA-256                   :",
    FD001_FD004_MANUSCRIPT_TABLE1_FILE_SHA256_631J,
)

print(
    "Table 2 CSV                       :",
    fd004_table_path_631j,
)

print(
    "Table 2 SHA-256                   :",
    FD004_MANUSCRIPT_TABLE2_FILE_SHA256_631J,
)

print(
    "Reporting JSON                    :",
    reporting_json_path_631j,
)

print(
    "Reporting JSON SHA-256            :",
    FD001_FD004_MANUSCRIPT_JSON_FILE_SHA256_631J,
)

print(
    "Manuscript Markdown               :",
    reporting_md_path_631j,
)

print(
    "Markdown SHA-256                  :",
    FD001_FD004_MANUSCRIPT_MD_FILE_SHA256_631J,
)

print(
    "Checkpoint                        :",
    checkpoint_path_631j,
)

print(
    "Checkpoint SHA-256                :",
    FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631J,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_631j,
)


print("\n--- WRITING STATE ---")

print(
    "Reporting bundle locked           :",
    FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J,
)

print(
    "Ready for Results chapter         :",
    FD001_FD004_READY_FOR_RESULTS_CHAPTER_631J,
)

print(
    "Ready for Discussion chapter      :",
    FD001_FD004_READY_FOR_DISCUSSION_CHAPTER_631J,
)

print(
    "Ready for Limitations section     :",
    FD001_FD004_READY_FOR_LIMITATIONS_SECTION_631J,
)


print(
    "\nPASS — manuscript-ready reporting evidence is frozen."
)

print(
    "No further scientific execution is required. "
    "Subsequent work is prose, tables, figures, and thesis/paper assembly only."
)

In [ ]:
# STEP 6.31J-R1 — REPORTING CHECKPOINT JSON-SEMANTIC ROUNDTRIP REPAIR
#
# Purpose:
# Repair ONLY the persistence/roundtrip check from STEP 6.31J.
#
# Cause of failure:
# Python tuples in the resident reporting payload are serialized as JSON arrays.
# json.load() restores those arrays as Python lists, so direct Python equality:
#
#     checkpoint_roundtrip == checkpoint_payload
#
# can fail even though the JSON content is semantically identical.
#
# This cell:
# - does NOT rebuild manuscript tables
# - does NOT rewrite scientific conclusions
# - does NOT recompute metrics
# - does NOT access TEST/RUL data
# - does NOT perform inference
# - verifies resident 6.31J objects
# - writes a JSON-normalized checkpoint
# - verifies exact JSON-semantic roundtrip
# - computes the file hashes that 6.31J did not reach
# - freezes the final reporting state

import os
import json
import hashlib
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS — 6.31J MUST HAVE REACHED REPORTING ARTIFACT CREATION
# =============================================================================

required_631jr1 = (
    # Frozen source science
    "all_checks_passed_631i",
    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I",
    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I",

    # 6.31J resident reporting objects created before failure
    "FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J",
    "FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J",
    "FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J",
    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J",
    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J",

    # 6.31J output paths created before failure
    "heldout_table_path_631j",
    "fd004_table_path_631j",
    "reporting_json_path_631j",
    "reporting_md_path_631j",
    "checkpoint_path_631j",

    # Original checkpoint payload constructed before failure
    "checkpoint_payload_631j",
)

missing_631jr1 = [
    name
    for name in required_631jr1
    if name not in globals()
]

if missing_631jr1:
    raise RuntimeError(
        "Missing STEP 6.31J-R1 resident object(s): "
        + ", ".join(missing_631jr1)
    )


if not bool(all_checks_passed_631i):
    raise RuntimeError(
        "STEP 6.31I must remain passed."
    )


# =============================================================================
# 1) HASH HELPERS
# =============================================================================

def _json_sha256_631jr1(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def _sha_file_631jr1(path):
    path = Path(
        path
    )

    if not path.is_file():
        raise RuntimeError(
            "Expected 6.31J reporting artifact is missing: "
            + str(path)
        )

    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


# =============================================================================
# 2) VERIFY 6.31J REPORTING CONTRACT WAS ALREADY CREATED CORRECTLY
# =============================================================================

reporting_contract_sha_recomputed_631jr1 = (
    _json_sha256_631jr1(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J
    )
)

if (
    reporting_contract_sha_recomputed_631jr1
    !=
    str(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J
    )
):
    raise RuntimeError(
        "Resident 6.31J reporting contract changed before persistence repair."
    )


if (
    str(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J[
            "source_synthesis_sha256"
        ]
    )
    !=
    str(
        FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I
    )
):
    raise RuntimeError(
        "6.31J reporting contract no longer points to frozen 6.31I synthesis."
    )


if (
    str(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J[
            "source_provenance_sha256"
        ]
    )
    !=
    str(
        FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I
    )
):
    raise RuntimeError(
        "6.31J reporting contract no longer points to frozen 6.31I provenance."
    )


# =============================================================================
# 3) CONFIRM REPORTING ARTIFACTS ALREADY EXIST
# =============================================================================

heldout_table_path_631jr1 = Path(
    heldout_table_path_631j
)

fd004_table_path_631jr1 = Path(
    fd004_table_path_631j
)

reporting_json_path_631jr1 = Path(
    reporting_json_path_631j
)

reporting_md_path_631jr1 = Path(
    reporting_md_path_631j
)

checkpoint_path_631jr1 = Path(
    checkpoint_path_631j
)


for path_631jr1 in (
    heldout_table_path_631jr1,
    fd004_table_path_631jr1,
    reporting_json_path_631jr1,
    reporting_md_path_631jr1,
):

    if not path_631jr1.is_file():
        raise RuntimeError(
            "6.31J failed before expected reporting artifact creation: "
            + str(
                path_631jr1
            )
        )


# =============================================================================
# 4) EXPLAIN / PROVE ORIGINAL DIRECT-EQUALITY FAILURE
# =============================================================================

# Convert the resident Python payload through JSON once.
# This is the canonical JSON-semantic representation.
checkpoint_payload_json_normalized_631jr1 = json.loads(
    json.dumps(
        checkpoint_payload_631j,
        sort_keys=True,
        allow_nan=False,
        default=str,
    )
)


# The original direct comparison may be False because tuple != list.
original_python_equality_631jr1 = bool(
    globals().get(
        "checkpoint_roundtrip_631j",
        None
    )
    ==
    checkpoint_payload_631j
)


# JSON-semantic hashes MUST be identical.
resident_payload_semantic_sha_631jr1 = (
    _json_sha256_631jr1(
        checkpoint_payload_631j
    )
)

normalized_payload_semantic_sha_631jr1 = (
    _json_sha256_631jr1(
        checkpoint_payload_json_normalized_631jr1
    )
)


if (
    resident_payload_semantic_sha_631jr1
    !=
    normalized_payload_semantic_sha_631jr1
):
    raise RuntimeError(
        "JSON normalization changed 6.31J checkpoint semantics."
    )


# =============================================================================
# 5) WRITE RESTART-SAFE JSON-NORMALIZED CHECKPOINT
# =============================================================================

safe_checkpoint_path_631jr1 = (
    checkpoint_path_631jr1.parent
    /
    "fd001_fd004_manuscript_reporting_631j_v2_safe.json"
)


tmp_safe_checkpoint_path_631jr1 = Path(
    str(
        safe_checkpoint_path_631jr1
    )
    +
    ".tmp"
)


with open(
    tmp_safe_checkpoint_path_631jr1,
    "w",
    encoding="utf-8",
) as handle_631jr1:

    json.dump(
        checkpoint_payload_json_normalized_631jr1,
        handle_631jr1,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


os.replace(
    tmp_safe_checkpoint_path_631jr1,
    safe_checkpoint_path_631jr1,
)


# =============================================================================
# 6) JSON-SEMANTIC ROUNDTRIP VERIFICATION
# =============================================================================

with open(
    safe_checkpoint_path_631jr1,
    "r",
    encoding="utf-8",
) as handle_631jr1:

    safe_roundtrip_631jr1 = json.load(
        handle_631jr1
    )


safe_roundtrip_exact_631jr1 = bool(
    safe_roundtrip_631jr1
    ==
    checkpoint_payload_json_normalized_631jr1
)


if not safe_roundtrip_exact_631jr1:
    raise RuntimeError(
        "6.31J-R1 normalized checkpoint roundtrip failed."
    )


safe_roundtrip_semantic_sha_631jr1 = (
    _json_sha256_631jr1(
        safe_roundtrip_631jr1
    )
)


if (
    safe_roundtrip_semantic_sha_631jr1
    !=
    resident_payload_semantic_sha_631jr1
):
    raise RuntimeError(
        "6.31J-R1 checkpoint semantic SHA differs from resident payload."
    )


# =============================================================================
# 7) COMPUTE REPORTING FILE HASHES THAT 6.31J DID NOT REACH
# =============================================================================

FD001_FD004_MANUSCRIPT_TABLE1_FILE_SHA256_631J = (
    _sha_file_631jr1(
        heldout_table_path_631jr1
    )
)

FD004_MANUSCRIPT_TABLE2_FILE_SHA256_631J = (
    _sha_file_631jr1(
        fd004_table_path_631jr1
    )
)

FD001_FD004_MANUSCRIPT_JSON_FILE_SHA256_631J = (
    _sha_file_631jr1(
        reporting_json_path_631jr1
    )
)

FD001_FD004_MANUSCRIPT_MD_FILE_SHA256_631J = (
    _sha_file_631jr1(
        reporting_md_path_631jr1
    )
)

FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631JR1 = (
    _sha_file_631jr1(
        safe_checkpoint_path_631jr1
    )
)


# =============================================================================
# 8) FREEZE PERSISTENCE-REPAIR AUDIT
# =============================================================================

FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_631JR1 = {
    "stage":
        "6.31J-R1",

    "status":
        "REPORTING_CHECKPOINT_JSON_SEMANTIC_ROUNDTRIP_REPAIRED",

    "original_failure_type":
        "PYTHON_TUPLE_LIST_JSON_ROUNDTRIP_EQUALITY_MISMATCH",

    "original_failure_scientific":
        False,

    "scientific_recomputation":
        False,

    "reporting_recomputation":
        False,

    "manuscript_tables_rebuilt":
        False,

    "reporting_bundle_rebuilt":
        False,

    "source_631i_synthesis_sha256":
        str(
            FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I
        ),

    "source_631i_provenance_sha256":
        str(
            FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I
        ),

    "reporting_contract_sha256":
        str(
            FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J
        ),

    "resident_checkpoint_semantic_sha256":
        resident_payload_semantic_sha_631jr1,

    "normalized_checkpoint_semantic_sha256":
        normalized_payload_semantic_sha_631jr1,

    "roundtrip_semantic_sha256":
        safe_roundtrip_semantic_sha_631jr1,

    "safe_checkpoint_path":
        str(
            safe_checkpoint_path_631jr1
        ),

    "safe_checkpoint_file_sha256":
        FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631JR1,

    "json_semantic_roundtrip_exact":
        safe_roundtrip_exact_631jr1,

    "raw_test_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_selection":
        False,

    "reporting_only":
        True,
}


FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_SHA256_631JR1 = (
    _json_sha256_631jr1(
        FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_631JR1
    )
)


# =============================================================================
# 9) FINAL REPORTING LOCK FLAGS
# =============================================================================

FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J = True
FD001_FD004_READY_FOR_RESULTS_CHAPTER_631J = True
FD001_FD004_READY_FOR_DISCUSSION_CHAPTER_631J = True
FD001_FD004_READY_FOR_LIMITATIONS_SECTION_631J = True

FD001_FD004_MANUSCRIPT_CHECKPOINT_REPAIRED_631JR1 = True

all_checks_passed_631j = True
all_checks_passed_631jr1 = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31J-R1 — REPORTING CHECKPOINT "
    "JSON-SEMANTIC ROUNDTRIP REPAIR"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Reporting bundle recomputed       :", False)
print("Manuscript tables rebuilt         :", False)
print("Raw TEST/RUL access               :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)


print("\n--- FAILURE DIAGNOSIS ---")

print(
    "Original direct Python equality   :",
    original_python_equality_631jr1,
)

print(
    "Failure cause                     :",
    "tuple/list conversion under JSON serialization",
)

print(
    "Scientific failure                :",
    False,
)


print("\n--- SEMANTIC INTEGRITY ---")

print(
    "Resident payload semantic SHA     :",
    resident_payload_semantic_sha_631jr1,
)

print(
    "Normalized payload semantic SHA   :",
    normalized_payload_semantic_sha_631jr1,
)

print(
    "Roundtrip semantic SHA            :",
    safe_roundtrip_semantic_sha_631jr1,
)

print(
    "Semantic SHA unchanged            :",
    bool(
        resident_payload_semantic_sha_631jr1
        ==
        normalized_payload_semantic_sha_631jr1
        ==
        safe_roundtrip_semantic_sha_631jr1
    ),
)

print(
    "JSON-semantic roundtrip exact     :",
    safe_roundtrip_exact_631jr1,
)


print("\n--- EXISTING 6.31J REPORTING ARTIFACTS ---")

print(
    "Table 1 CSV                       :",
    heldout_table_path_631jr1,
)

print(
    "Table 1 SHA-256                   :",
    FD001_FD004_MANUSCRIPT_TABLE1_FILE_SHA256_631J,
)

print(
    "Table 2 CSV                       :",
    fd004_table_path_631jr1,
)

print(
    "Table 2 SHA-256                   :",
    FD004_MANUSCRIPT_TABLE2_FILE_SHA256_631J,
)

print(
    "Reporting JSON                    :",
    reporting_json_path_631jr1,
)

print(
    "Reporting JSON SHA-256            :",
    FD001_FD004_MANUSCRIPT_JSON_FILE_SHA256_631J,
)

print(
    "Manuscript Markdown               :",
    reporting_md_path_631jr1,
)

print(
    "Markdown SHA-256                  :",
    FD001_FD004_MANUSCRIPT_MD_FILE_SHA256_631J,
)


print("\n--- REPAIRED CHECKPOINT ---")

print(
    "Safe checkpoint                   :",
    safe_checkpoint_path_631jr1,
)

print(
    "Safe checkpoint SHA-256           :",
    FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631JR1,
)

print(
    "Persistence repair SHA-256        :",
    FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_SHA256_631JR1,
)


print("\n--- FINAL WRITING STATE ---")

print(
    "Reporting bundle locked           :",
    FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J,
)

print(
    "Checkpoint repaired               :",
    FD001_FD004_MANUSCRIPT_CHECKPOINT_REPAIRED_631JR1,
)

print(
    "Ready for Results chapter         :",
    FD001_FD004_READY_FOR_RESULTS_CHAPTER_631J,
)

print(
    "Ready for Discussion chapter      :",
    FD001_FD004_READY_FOR_DISCUSSION_CHAPTER_631J,
)

print(
    "Ready for Limitations section     :",
    FD001_FD004_READY_FOR_LIMITATIONS_SECTION_631J,
)


print(
    "\nPASS — 6.31J scientific/reporting content was unchanged; "
    "only JSON checkpoint persistence was repaired."
)

print(
    "The empirical and reporting-evidence phases are now closed. "
    "Subsequent work is manuscript prose, figures, formatting, and assembly only."
)

In [ ]:
# STEP 6.31K — MANUSCRIPT FIGURE + CAPTION EXPORT
#
# Purpose:
# Create manuscript-ready visual reporting artifacts from the already-frozen
# 6.31J reporting tables only.
#
# Figures:
#   Figure 1 — held-out selective MAE effect by dataset/target
#              (gain / loss relative to full-population MAE)
#
#   Figure 2 — FD004 controlled-corruption selective-gain outcomes
#
# IMPORTANT SCIENTIFIC GUARDS:
# - No raw prediction/label arrays are used.
# - No metric is recomputed from predictions.
# - No raw TEST/RUL files are read.
# - Absolute MAE values are NOT plotted across datasets.
# - Capped and uncapped targets remain explicitly labeled.
# - FD004 TEST is shown as unscored, not as zero performance.
# - FD004 DEV/corruption evidence remains separate from held-out TEST evidence.
#
# REPORTING / PRESENTATION ONLY.

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631k = (
    "all_checks_passed_631jr1",

    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J",
    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J",

    "FD001_FD004_MANUSCRIPT_REPORTING_BUNDLE_631J",
    "FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J",
    "FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J",

    "FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J",
    "FD001_FD004_MANUSCRIPT_CHECKPOINT_REPAIRED_631JR1",
)

missing_631k = [
    name
    for name in required_631k
    if name not in globals()
]

if missing_631k:
    raise RuntimeError(
        "Missing STEP 6.31K authority object(s): "
        + ", ".join(missing_631k)
    )


if not bool(all_checks_passed_631jr1):
    raise RuntimeError(
        "STEP 6.31J-R1 must remain passed."
    )


if not bool(
    FD001_FD004_MANUSCRIPT_REPORTING_LOCKED_631J
):
    raise RuntimeError(
        "6.31J reporting bundle is not locked."
    )


if not bool(
    FD001_FD004_MANUSCRIPT_CHECKPOINT_REPAIRED_631JR1
):
    raise RuntimeError(
        "6.31J persistence repair is not locked."
    )


# =============================================================================
# 1) VERIFY FROZEN REPORTING CONTRACT
# =============================================================================

def _json_sha256_631k(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha256_631k(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_631J
    )
    !=
    str(
        FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J
    )
):
    raise RuntimeError(
        "Frozen 6.31J reporting contract changed."
    )


# =============================================================================
# 2) BIND FROZEN TABLES
# =============================================================================

heldout_table_631k = (
    FD001_FD004_MANUSCRIPT_HELDOUT_TABLE_631J.copy(
        deep=True
    )
)

fd004_support_table_631k = (
    FD004_MANUSCRIPT_SUPPORTING_EVIDENCE_TABLE_631J.copy(
        deep=True
    )
)


if len(
    heldout_table_631k
) != 5:
    raise RuntimeError(
        "Expected five frozen target-aware held-out rows."
    )


# =============================================================================
# 3) OUTPUT DIRECTORY
# =============================================================================

figure_dir_631k = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\cross_dataset_synthesis"
    r"\manuscript_ready\figures"
)

figure_dir_631k.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 4) FIGURE 1 DATA
#
# Use selective MAE gain only.
#
# Gain:
#     positive -> accepted subset has lower MAE than full population
#     negative -> accepted subset has higher MAE than full population
#
# This avoids an invalid absolute-MAE cross-dataset ranking.
# =============================================================================

figure1_data_631k = heldout_table_631k[
    [
        "Dataset",
        "Target",
        "Analysis role",
        "Coverage",
        "Selective MAE gain",
        "Relative MAE reduction (%)",
        "Held-out status",
    ]
].copy()


figure1_data_631k[
    "Plot label"
] = figure1_data_631k.apply(
    lambda row: (
        f"{row['Dataset']}\n"
        f"{row['Target']}"
    ),
    axis=1,
)


scored_mask_631k = (
    figure1_data_631k[
        "Held-out status"
    ]
    ==
    "Scored"
)


scored_figure1_631k = (
    figure1_data_631k.loc[
        scored_mask_631k
    ].copy()
)


unscored_figure1_631k = (
    figure1_data_631k.loc[
        ~scored_mask_631k
    ].copy()
)


# =============================================================================
# 5) FIGURE 1 — HELD-OUT SELECTIVE MAE EFFECT
# =============================================================================

fig1_631k, ax1_631k = plt.subplots(
    figsize=(
        10.5,
        6.2,
    )
)


x1_631k = np.arange(
    len(
        scored_figure1_631k
    )
)


gain1_631k = scored_figure1_631k[
    "Selective MAE gain"
].to_numpy(
    dtype=float
)


bars1_631k = ax1_631k.bar(
    x1_631k,
    gain1_631k,
)


ax1_631k.axhline(
    0.0,
    linewidth=1.0,
)


ax1_631k.set_xticks(
    x1_631k
)

ax1_631k.set_xticklabels(
    scored_figure1_631k[
        "Plot label"
    ].tolist(),
    rotation=0,
)


ax1_631k.set_ylabel(
    "Selective MAE gain\n(full MAE − accepted MAE)"
)

ax1_631k.set_title(
    "Held-out selective MAE effect under frozen reliability policies"
)


# Label each bar with gain and realized coverage.
for idx_631k, (
    bar_631k,
    (_, row_631k),
) in enumerate(
    zip(
        bars1_631k,
        scored_figure1_631k.iterrows(),
    )
):

    gain_631k = float(
        row_631k[
            "Selective MAE gain"
        ]
    )

    coverage_631k = row_631k[
        "Coverage"
    ]

    coverage_text_631k = (
        "coverage n/a"
        if pd.isna(
            coverage_631k
        )
        else
        f"coverage {100.0 * float(coverage_631k):.1f}%"
    )

    relative_631k = row_631k[
        "Relative MAE reduction (%)"
    ]

    relative_text_631k = (
        ""
        if pd.isna(
            relative_631k
        )
        else
        f"\n{float(relative_631k):+.2f}% relative"
    )


    vertical_offset_631k = (
        6
        if gain_631k >= 0
        else
        -20
    )


    ax1_631k.annotate(
        (
            f"{gain_631k:+.3f}\n"
            f"{coverage_text_631k}"
            f"{relative_text_631k}"
        ),
        xy=(
            bar_631k.get_x()
            +
            bar_631k.get_width()
            /
            2.0,
            gain_631k,
        ),
        xytext=(
            0,
            vertical_offset_631k,
        ),
        textcoords="offset points",
        ha="center",
        va=(
            "bottom"
            if gain_631k >= 0
            else
            "top"
        ),
        fontsize=8.5,
    )


# Explicitly state FD004's held-out status rather than plotting NaN as zero.
if len(
    unscored_figure1_631k
) > 0:

    unscored_text_631k = "; ".join(
        (
            f"{row['Dataset']} {row['Target']}: "
            f"{row['Held-out status']}"
        )
        for _, row
        in unscored_figure1_631k.iterrows()
    )

    ax1_631k.text(
        0.01,
        -0.24,
        (
            "Not represented as a performance bar: "
            +
            unscored_text_631k
        ),
        transform=ax1_631k.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        wrap=True,
    )


ax1_631k.text(
    0.01,
    -0.34,
    (
        "Positive values indicate lower MAE among accepted predictions. "
        "Targets differ across rows; this figure is not an absolute-MAE "
        "cross-dataset ranking."
    ),
    transform=ax1_631k.transAxes,
    ha="left",
    va="top",
    fontsize=8.5,
    wrap=True,
)


fig1_631k.tight_layout()


figure1_png_631k = (
    figure_dir_631k
    /
    "figure1_heldout_selective_mae_effect_631k.png"
)

figure1_pdf_631k = (
    figure_dir_631k
    /
    "figure1_heldout_selective_mae_effect_631k.pdf"
)


fig1_631k.savefig(
    figure1_png_631k,
    dpi=300,
    bbox_inches="tight",
)

fig1_631k.savefig(
    figure1_pdf_631k,
    bbox_inches="tight",
)

plt.close(
    fig1_631k
)


# =============================================================================
# 6) FIGURE 2 DATA — FD004 CONTROLLED CORRUPTION
# =============================================================================

def _lookup_support_value_631k(
    block,
    measure,
):
    rows = fd004_support_table_631k[
        (
            fd004_support_table_631k[
                "Evidence block"
            ]
            ==
            block
        )
        &
        (
            fd004_support_table_631k[
                "Measure"
            ]
            ==
            measure
        )
    ]

    if len(
        rows
    ) != 1:
        raise RuntimeError(
            f"Expected exactly one FD004 support row: "
            f"{block} / {measure}"
        )

    return rows.iloc[
        0
    ][
        "Value"
    ]


fd004_positive_count_631k = int(
    float(
        _lookup_support_value_631k(
            "Controlled corruption",
            "Positive-gain scenarios",
        )
    )
)

fd004_zero_count_631k = int(
    float(
        _lookup_support_value_631k(
            "Controlled corruption",
            "Zero-gain scenarios",
        )
    )
)

fd004_negative_count_631k = int(
    float(
        _lookup_support_value_631k(
            "Controlled corruption",
            "Negative-gain scenarios",
        )
    )
)


fd004_scenario_count_631k = int(
    float(
        _lookup_support_value_631k(
            "Controlled corruption",
            "Executable scenarios",
        )
    )
)


if (
    fd004_positive_count_631k
    +
    fd004_zero_count_631k
    +
    fd004_negative_count_631k
    !=
    fd004_scenario_count_631k
):
    raise RuntimeError(
        "FD004 corruption outcome counts do not sum to scenario count."
    )


# =============================================================================
# 7) FIGURE 2 — FD004 CORRUPTION OUTCOMES
# =============================================================================

fig2_631k, ax2_631k = plt.subplots(
    figsize=(
        8.2,
        5.8,
    )
)


categories2_631k = [
    "Positive gain",
    "Zero gain",
    "Negative gain",
]

counts2_631k = [
    fd004_positive_count_631k,
    fd004_zero_count_631k,
    fd004_negative_count_631k,
]


x2_631k = np.arange(
    len(
        categories2_631k
    )
)


bars2_631k = ax2_631k.bar(
    x2_631k,
    counts2_631k,
)


ax2_631k.set_xticks(
    x2_631k
)

ax2_631k.set_xticklabels(
    categories2_631k
)

ax2_631k.set_ylabel(
    "Controlled-degradation scenarios"
)

ax2_631k.set_title(
    "FD004 selective-gain direction across controlled sensor degradation"
)


for bar_631k, count_631k in zip(
    bars2_631k,
    counts2_631k,
):

    pct_631k = (
        100.0
        *
        count_631k
        /
        fd004_scenario_count_631k
    )

    ax2_631k.annotate(
        (
            f"{count_631k}\n"
            f"({pct_631k:.1f}%)"
        ),
        xy=(
            bar_631k.get_x()
            +
            bar_631k.get_width()
            /
            2.0,
            count_631k,
        ),
        xytext=(
            0,
            5,
        ),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )


ax2_631k.text(
    0.01,
    -0.20,
    (
        "All frozen negative-gain scenarios occurred in the stuck-sensor "
        "family; these DEV/corruption results are supporting evidence and "
        "are not held-out TEST performance."
    ),
    transform=ax2_631k.transAxes,
    ha="left",
    va="top",
    fontsize=8.5,
    wrap=True,
)


fig2_631k.tight_layout()


figure2_png_631k = (
    figure_dir_631k
    /
    "figure2_fd004_corruption_gain_direction_631k.png"
)

figure2_pdf_631k = (
    figure_dir_631k
    /
    "figure2_fd004_corruption_gain_direction_631k.pdf"
)


fig2_631k.savefig(
    figure2_png_631k,
    dpi=300,
    bbox_inches="tight",
)

fig2_631k.savefig(
    figure2_pdf_631k,
    bbox_inches="tight",
)

plt.close(
    fig2_631k
)


# =============================================================================
# 8) MANUSCRIPT CAPTIONS
# =============================================================================

FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K = {
    "Figure 1":
        (
            "Held-out selective MAE effect under the frozen reliability "
            "policies. Selective MAE gain is defined as full-population MAE "
            "minus accepted-set MAE, so positive values indicate lower error "
            "among accepted predictions. FD001 is shown separately for its "
            "primary capped-RUL target and retained raw-RUL secondary "
            "analysis; FD002 uses the frozen piecewise-capped RUL target; "
            "FD003 uses uncapped RUL. FD004 is not represented as a "
            "performance bar because its primary held-out TEST evaluation "
            "was stopped before inference when 11 of 248 trajectories were "
            "shorter than the frozen 30-cycle input requirement. The figure "
            "does not imply comparability of absolute MAE across datasets."
        ),

    "Figure 2":
        (
            "FD004 selective-gain direction across 381 executable controlled "
            "sensor-degradation scenarios. Positive selective gain occurred "
            "in 365 scenarios, zero gain in none, and negative gain in 16. "
            "All negative-gain cases belonged to the stuck-sensor family. "
            "These results are supporting DEV/corruption evidence and must "
            "not be interpreted as held-out TEST performance."
        ),
}


# =============================================================================
# 9) FIGURE REPORTING CONTRACT
# =============================================================================

FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K = {
    "stage":
        "6.31K",

    "status":
        "MANUSCRIPT_FIGURES_AND_CAPTIONS_COMPLETE",

    "source_reporting_contract_sha256":
        str(
            FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J
        ),

    "figure_1":
        {
            "scientific_quantity":
                "held-out selective MAE gain",

            "absolute_mae_cross_dataset_ranking":
                False,

            "fd004_unscored_test_plotted_as_zero":
                False,

            "target_labels_explicit":
                True,
        },

    "figure_2":
        {
            "scientific_quantity":
                "FD004 controlled-corruption selective-gain direction",

            "evaluation_stage":
                "DEV / controlled corruption",

            "presented_as_heldout_test":
                False,
        },

    "scientific_recomputation":
        False,

    "metric_recomputation_from_predictions":
        False,

    "raw_test_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_selection":
        False,

    "reporting_only":
        True,
}


FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K = (
    _json_sha256_631k(
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K
    )
)


# =============================================================================
# 10) WRITE CAPTION / METADATA ARTIFACT
# =============================================================================

figure_metadata_path_631k = (
    figure_dir_631k
    /
    "figure_captions_and_reporting_contract_631k.json"
)


figure_metadata_payload_631k = {
    "captions":
        FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K,

    "figure_contract":
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K,

    "figure_contract_sha256":
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K,
}


with open(
    figure_metadata_path_631k,
    "w",
    encoding="utf-8",
) as handle_631k:

    json.dump(
        figure_metadata_payload_631k,
        handle_631k,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


# =============================================================================
# 11) HASH OUTPUT FILES
# =============================================================================

def _sha_file_631k(path):
    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


FD001_FD004_FIGURE1_PNG_SHA256_631K = (
    _sha_file_631k(
        figure1_png_631k
    )
)

FD001_FD004_FIGURE1_PDF_SHA256_631K = (
    _sha_file_631k(
        figure1_pdf_631k
    )
)

FD004_FIGURE2_PNG_SHA256_631K = (
    _sha_file_631k(
        figure2_png_631k
    )
)

FD004_FIGURE2_PDF_SHA256_631K = (
    _sha_file_631k(
        figure2_pdf_631k
    )
)

FD001_FD004_FIGURE_METADATA_SHA256_631K = (
    _sha_file_631k(
        figure_metadata_path_631k
    )
)


# =============================================================================
# 12) RESTART-SAFE FIGURE CHECKPOINT
# =============================================================================

checkpoint_payload_631k = {
    "figure_contract":
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K,

    "figure_contract_sha256":
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K,

    "captions":
        FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K,

    "files":
        {
            "figure1_png":
                str(
                    figure1_png_631k
                ),

            "figure1_png_sha256":
                FD001_FD004_FIGURE1_PNG_SHA256_631K,

            "figure1_pdf":
                str(
                    figure1_pdf_631k
                ),

            "figure1_pdf_sha256":
                FD001_FD004_FIGURE1_PDF_SHA256_631K,

            "figure2_png":
                str(
                    figure2_png_631k
                ),

            "figure2_png_sha256":
                FD004_FIGURE2_PNG_SHA256_631K,

            "figure2_pdf":
                str(
                    figure2_pdf_631k
                ),

            "figure2_pdf_sha256":
                FD004_FIGURE2_PDF_SHA256_631K,

            "metadata_json":
                str(
                    figure_metadata_path_631k
                ),

            "metadata_json_sha256":
                FD001_FD004_FIGURE_METADATA_SHA256_631K,
        },
}


checkpoint_path_631k = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd001_fd004_manuscript_figures_631k_v1.json"
)


checkpoint_path_631k.parent.mkdir(
    parents=True,
    exist_ok=True,
)


with open(
    checkpoint_path_631k,
    "w",
    encoding="utf-8",
) as handle_631k:

    json.dump(
        checkpoint_payload_631k,
        handle_631k,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


with open(
    checkpoint_path_631k,
    "r",
    encoding="utf-8",
) as handle_631k:

    checkpoint_roundtrip_631k = json.load(
        handle_631k
    )


checkpoint_roundtrip_exact_631k = bool(
    checkpoint_roundtrip_631k
    ==
    checkpoint_payload_631k
)


if not checkpoint_roundtrip_exact_631k:
    raise RuntimeError(
        "6.31K figure checkpoint roundtrip failed."
    )


FD001_FD004_FIGURE_CHECKPOINT_FILE_SHA256_631K = (
    _sha_file_631k(
        checkpoint_path_631k
    )
)


# =============================================================================
# 13) FINAL FLAGS
# =============================================================================

FD001_FD004_MANUSCRIPT_FIGURES_LOCKED_631K = True
FD001_FD004_READY_FOR_MANUSCRIPT_ASSEMBLY_631K = True

all_checks_passed_631k = True


# =============================================================================
# 14) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31K — MANUSCRIPT FIGURE + CAPTION EXPORT"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST/RUL access               :", False)
print("Prediction-level metrics used     :", False)
print("Absolute-MAE dataset ranking      :", False)
print("Reporting-only                    :", True)


print("\n--- FIGURE 1 ---")

print(
    "PNG                               :",
    figure1_png_631k,
)

print(
    "PNG SHA-256                       :",
    FD001_FD004_FIGURE1_PNG_SHA256_631K,
)

print(
    "PDF                               :",
    figure1_pdf_631k,
)

print(
    "PDF SHA-256                       :",
    FD001_FD004_FIGURE1_PDF_SHA256_631K,
)

print(
    "Caption:"
)

print(
    FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K[
        "Figure 1"
    ]
)


print("\n--- FIGURE 2 ---")

print(
    "PNG                               :",
    figure2_png_631k,
)

print(
    "PNG SHA-256                       :",
    FD004_FIGURE2_PNG_SHA256_631K,
)

print(
    "PDF                               :",
    figure2_pdf_631k,
)

print(
    "PDF SHA-256                       :",
    FD004_FIGURE2_PDF_SHA256_631K,
)

print(
    "Caption:"
)

print(
    FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K[
        "Figure 2"
    ]
)


print("\n--- METADATA / CHECKPOINT ---")

print(
    "Figure metadata JSON              :",
    figure_metadata_path_631k,
)

print(
    "Metadata SHA-256                  :",
    FD001_FD004_FIGURE_METADATA_SHA256_631K,
)

print(
    "Figure contract SHA-256           :",
    FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K,
)

print(
    "Checkpoint                        :",
    checkpoint_path_631k,
)

print(
    "Checkpoint SHA-256                :",
    FD001_FD004_FIGURE_CHECKPOINT_FILE_SHA256_631K,
)

print(
    "Checkpoint roundtrip exact        :",
    checkpoint_roundtrip_exact_631k,
)


print("\n--- FINAL STATE ---")

print(
    "Figures locked                    :",
    FD001_FD004_MANUSCRIPT_FIGURES_LOCKED_631K,
)

print(
    "Ready for manuscript assembly     :",
    FD001_FD004_READY_FOR_MANUSCRIPT_ASSEMBLY_631K,
)


print(
    "\nPASS — manuscript figures and captions were generated "
    "from frozen reporting evidence only."
)

print(
    "No further notebook science is required after this step."
)

In [ ]:
# STEP 6.31L — FINAL MANUSCRIPT-ASSEMBLY MANIFEST + PROJECT CLOSURE
#
# Purpose:
# Create one restart-safe final manifest for all frozen scientific/reporting
# artifacts produced through STEP 6.31K.
#
# This is the FINAL notebook bookkeeping step.
#
# NO science.
# NO metric recomputation.
# NO predictions.
# NO labels.
# NO TEST/RUL access.
# NO inference.
# NO plotting.
# NO manuscript prose generation.
#
# After this step, the notebook workflow is CLOSED.

import os
import json
import hashlib
from pathlib import Path


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631l = (
    "all_checks_passed_631k",

    "FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I",
    "FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I",

    "FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J",
    "FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_SHA256_631JR1",

    "FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K",
    "FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K",
    "FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K",

    "FD001_FD004_MANUSCRIPT_FIGURES_LOCKED_631K",
    "FD001_FD004_READY_FOR_MANUSCRIPT_ASSEMBLY_631K",

    # Reporting artifact paths
    "heldout_table_path_631j",
    "fd004_table_path_631j",
    "reporting_json_path_631j",
    "reporting_md_path_631j",
    "safe_checkpoint_path_631jr1",

    # Figure artifact paths
    "figure1_png_631k",
    "figure1_pdf_631k",
    "figure2_png_631k",
    "figure2_pdf_631k",
    "figure_metadata_path_631k",
    "checkpoint_path_631k",
)

missing_631l = [
    name
    for name in required_631l
    if name not in globals()
]

if missing_631l:
    raise RuntimeError(
        "Missing STEP 6.31L authority object(s): "
        + ", ".join(missing_631l)
    )


if not bool(all_checks_passed_631k):
    raise RuntimeError(
        "STEP 6.31K must remain passed."
    )


if not bool(
    FD001_FD004_MANUSCRIPT_FIGURES_LOCKED_631K
):
    raise RuntimeError(
        "6.31K figures are not locked."
    )


if not bool(
    FD001_FD004_READY_FOR_MANUSCRIPT_ASSEMBLY_631K
):
    raise RuntimeError(
        "6.31K did not authorize manuscript assembly."
    )


# =============================================================================
# 1) HELPERS
# =============================================================================

def _json_sha256_631l(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def _file_sha256_631l(path):
    path = Path(
        path
    )

    if not path.is_file():
        raise RuntimeError(
            "Required final artifact missing: "
            + str(path)
        )

    with open(
        path,
        "rb",
    ) as handle:
        return hashlib.sha256(
            handle.read()
        ).hexdigest()


# =============================================================================
# 2) VERIFY FIGURE CONTRACT
# =============================================================================

if (
    _json_sha256_631l(
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_631K
    )
    !=
    str(
        FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K
    )
):
    raise RuntimeError(
        "Frozen 6.31K figure contract changed."
    )


# =============================================================================
# 3) BIND ALL FINAL ARTIFACT PATHS
# =============================================================================

artifact_paths_631l = {
    # 6.31J reporting
    "table1_target_aware_heldout_csv":
        Path(
            heldout_table_path_631j
        ),

    "table2_fd004_supporting_csv":
        Path(
            fd004_table_path_631j
        ),

    "reporting_bundle_json":
        Path(
            reporting_json_path_631j
        ),

    "reporting_bundle_markdown":
        Path(
            reporting_md_path_631j
        ),

    "reporting_checkpoint_safe":
        Path(
            safe_checkpoint_path_631jr1
        ),

    # 6.31K figures
    "figure1_png":
        Path(
            figure1_png_631k
        ),

    "figure1_pdf":
        Path(
            figure1_pdf_631k
        ),

    "figure2_png":
        Path(
            figure2_png_631k
        ),

    "figure2_pdf":
        Path(
            figure2_pdf_631k
        ),

    "figure_metadata_json":
        Path(
            figure_metadata_path_631k
        ),

    "figure_checkpoint":
        Path(
            checkpoint_path_631k
        ),
}


# =============================================================================
# 4) VERIFY ALL ARTIFACTS + HASH THEM
# =============================================================================

artifact_manifest_631l = {}


for key_631l, path_631l in artifact_paths_631l.items():

    if not path_631l.is_file():
        raise RuntimeError(
            "Final manuscript artifact missing: "
            + str(
                path_631l
            )
        )

    artifact_manifest_631l[
        key_631l
    ] = {
        "path":
            str(
                path_631l
            ),

        "size_bytes":
            int(
                path_631l.stat().st_size
            ),

        "sha256":
            _file_sha256_631l(
                path_631l
            ),
    }


# =============================================================================
# 5) VERIFY KNOWN HASHES FROM PRIOR LOCKED STEPS
# =============================================================================

known_hash_checks_631l = {
    "table1":
        (
            artifact_manifest_631l[
                "table1_target_aware_heldout_csv"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_MANUSCRIPT_TABLE1_FILE_SHA256_631J
            )
        ),

    "table2":
        (
            artifact_manifest_631l[
                "table2_fd004_supporting_csv"
            ][
                "sha256"
            ]
            ==
            str(
                FD004_MANUSCRIPT_TABLE2_FILE_SHA256_631J
            )
        ),

    "reporting_json":
        (
            artifact_manifest_631l[
                "reporting_bundle_json"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_MANUSCRIPT_JSON_FILE_SHA256_631J
            )
        ),

    "reporting_markdown":
        (
            artifact_manifest_631l[
                "reporting_bundle_markdown"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_MANUSCRIPT_MD_FILE_SHA256_631J
            )
        ),

    "reporting_checkpoint":
        (
            artifact_manifest_631l[
                "reporting_checkpoint_safe"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_MANUSCRIPT_CHECKPOINT_FILE_SHA256_631JR1
            )
        ),

    "figure1_png":
        (
            artifact_manifest_631l[
                "figure1_png"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_FIGURE1_PNG_SHA256_631K
            )
        ),

    "figure1_pdf":
        (
            artifact_manifest_631l[
                "figure1_pdf"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_FIGURE1_PDF_SHA256_631K
            )
        ),

    "figure2_png":
        (
            artifact_manifest_631l[
                "figure2_png"
            ][
                "sha256"
            ]
            ==
            str(
                FD004_FIGURE2_PNG_SHA256_631K
            )
        ),

    "figure2_pdf":
        (
            artifact_manifest_631l[
                "figure2_pdf"
            ][
                "sha256"
            ]
            ==
            str(
                FD004_FIGURE2_PDF_SHA256_631K
            )
        ),

    "figure_metadata":
        (
            artifact_manifest_631l[
                "figure_metadata_json"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_FIGURE_METADATA_SHA256_631K
            )
        ),

    "figure_checkpoint":
        (
            artifact_manifest_631l[
                "figure_checkpoint"
            ][
                "sha256"
            ]
            ==
            str(
                FD001_FD004_FIGURE_CHECKPOINT_FILE_SHA256_631K
            )
        ),
}


if not all(
    known_hash_checks_631l.values()
):
    failed_hashes_631l = [
        key
        for key, passed
        in known_hash_checks_631l.items()
        if not passed
    ]

    raise RuntimeError(
        "Final artifact hash mismatch(es): "
        + ", ".join(
            failed_hashes_631l
        )
    )


# =============================================================================
# 6) FINAL SCIENTIFIC / REPORTING STATE
# =============================================================================

final_state_631l = {
    "empirical_phase":
        "CLOSED",

    "scientific_execution":
        "CLOSED",

    "cross_dataset_synthesis":
        "LOCKED",

    "reporting_evidence":
        "LOCKED",

    "manuscript_tables":
        "LOCKED",

    "manuscript_figures":
        "LOCKED",

    "remaining_work":
        (
            "manuscript prose, thesis/paper formatting, "
            "citation integration, and final document assembly"
        ),

    "new_scientific_execution_permitted":
        False,

    "posthoc_test_retuning_permitted":
        False,

    "cross_dataset_absolute_mae_ranking_permitted":
        False,

    "fd004_heldout_test_metric_available":
        False,
}


# =============================================================================
# 7) FINAL MANUSCRIPT-ASSEMBLY MANIFEST
# =============================================================================

FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L = {
    "stage":
        "6.31L",

    "status":
        "FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_LOCKED",

    "project":
        (
            "Reliability-Aware Selective Remaining Useful Life "
            "Prediction under Sensor Degradation"
        ),

    "source_contracts": {
        "final_target_aware_synthesis_sha256":
            str(
                FD001_FD004_FINAL_TARGET_AWARE_SYNTHESIS_SHA256_631I
            ),

        "final_synthesis_provenance_sha256":
            str(
                FD001_FD004_FINAL_SYNTHESIS_PROVENANCE_SHA256_631I
            ),

        "manuscript_reporting_contract_sha256":
            str(
                FD001_FD004_MANUSCRIPT_REPORTING_CONTRACT_SHA256_631J
            ),

        "reporting_persistence_repair_sha256":
            str(
                FD001_FD004_MANUSCRIPT_PERSISTENCE_REPAIR_SHA256_631JR1
            ),

        "figure_contract_sha256":
            str(
                FD001_FD004_MANUSCRIPT_FIGURE_CONTRACT_SHA256_631K
            ),
    },

    "artifacts":
        artifact_manifest_631l,

    "figure_captions":
        FD001_FD004_MANUSCRIPT_FIGURE_CAPTIONS_631K,

    "final_state":
        final_state_631l,

    "scientific_recomputation":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "raw_test_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_policy_selection":
        False,

    "plot_regeneration":
        False,

    "reporting_only":
        True,
}


FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L = (
    _json_sha256_631l(
        FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L
    )
)


# =============================================================================
# 8) WRITE FINAL DELIVERY MANIFEST
#
# JSON-normalize before persistence so tuple/list semantics cannot trigger
# another false roundtrip failure.
# =============================================================================

final_output_dir_631l = Path(
    r"C:\RUL_Reliability_Research"
    r"\06_results\cross_dataset_synthesis"
    r"\manuscript_ready"
)


final_manifest_path_631l = (
    final_output_dir_631l
    /
    "FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L.json"
)


normalized_manifest_631l = json.loads(
    json.dumps(
        FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L,
        sort_keys=True,
        allow_nan=False,
        default=str,
    )
)


with open(
    final_manifest_path_631l,
    "w",
    encoding="utf-8",
) as handle_631l:

    json.dump(
        normalized_manifest_631l,
        handle_631l,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


# =============================================================================
# 9) ROUNDTRIP VERIFY FINAL MANIFEST
# =============================================================================

with open(
    final_manifest_path_631l,
    "r",
    encoding="utf-8",
) as handle_631l:

    final_manifest_roundtrip_631l = (
        json.load(
            handle_631l
        )
    )


final_manifest_roundtrip_exact_631l = bool(
    final_manifest_roundtrip_631l
    ==
    normalized_manifest_631l
)


if not final_manifest_roundtrip_exact_631l:
    raise RuntimeError(
        "Final manuscript manifest roundtrip failed."
    )


final_manifest_semantic_sha_631l = (
    _json_sha256_631l(
        final_manifest_roundtrip_631l
    )
)


if (
    final_manifest_semantic_sha_631l
    !=
    FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L
):
    raise RuntimeError(
        "Final manuscript manifest semantic SHA changed."
    )


FD001_FD004_FINAL_MANIFEST_FILE_SHA256_631L = (
    _file_sha256_631l(
        final_manifest_path_631l
    )
)


# =============================================================================
# 10) FINAL PROJECT-CLOSURE CHECKPOINT
# =============================================================================

closure_checkpoint_payload_631l = {
    "manifest":
        normalized_manifest_631l,

    "manifest_semantic_sha256":
        FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L,

    "manifest_file_sha256":
        FD001_FD004_FINAL_MANIFEST_FILE_SHA256_631L,
}


closure_checkpoint_path_631l = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd001_fd004_project_final_closure_631l_v1.json"
)


closure_checkpoint_path_631l.parent.mkdir(
    parents=True,
    exist_ok=True,
)


with open(
    closure_checkpoint_path_631l,
    "w",
    encoding="utf-8",
) as handle_631l:

    json.dump(
        closure_checkpoint_payload_631l,
        handle_631l,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )


with open(
    closure_checkpoint_path_631l,
    "r",
    encoding="utf-8",
) as handle_631l:

    closure_roundtrip_631l = (
        json.load(
            handle_631l
        )
    )


closure_roundtrip_exact_631l = bool(
    closure_roundtrip_631l
    ==
    closure_checkpoint_payload_631l
)


if not closure_roundtrip_exact_631l:
    raise RuntimeError(
        "Final project-closure checkpoint roundtrip failed."
    )


FD001_FD004_FINAL_CLOSURE_CHECKPOINT_SHA256_631L = (
    _file_sha256_631l(
        closure_checkpoint_path_631l
    )
)


# =============================================================================
# 11) FINAL PROJECT FLAGS
# =============================================================================

FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L = True
FD001_FD004_PROJECT_REPORTING_PHASE_CLOSED_631L = True
FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L = True

FD001_FD004_READY_FOR_EXTERNAL_MANUSCRIPT_ASSEMBLY_631L = True

all_checks_passed_631l = True


# =============================================================================
# 12) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31L — FINAL MANUSCRIPT-ASSEMBLY "
    "MANIFEST + PROJECT CLOSURE"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Raw TEST/RUL access               :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Plot regeneration                 :", False)
print("Reporting-only                    :", True)


print("\n--- FINAL ARTIFACT INVENTORY ---")

for key_631l, row_631l in artifact_manifest_631l.items():

    print(
        f"\n{key_631l}"
    )

    print(
        "  path                            :",
        row_631l[
            "path"
        ],
    )

    print(
        "  size bytes                      :",
        row_631l[
            "size_bytes"
        ],
    )

    print(
        "  SHA-256                         :",
        row_631l[
            "sha256"
        ],
    )


print("\n--- HASH VERIFICATION ---")

for key_631l, passed_631l in known_hash_checks_631l.items():

    print(
        f"{key_631l:35s}:",
        passed_631l,
    )


print("\n--- FINAL MANIFEST ---")

print(
    "Manifest path                     :",
    final_manifest_path_631l,
)

print(
    "Manifest semantic SHA-256         :",
    FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L,
)

print(
    "Manifest file SHA-256             :",
    FD001_FD004_FINAL_MANIFEST_FILE_SHA256_631L,
)

print(
    "Manifest roundtrip exact          :",
    final_manifest_roundtrip_exact_631l,
)


print("\n--- FINAL CLOSURE CHECKPOINT ---")

print(
    "Checkpoint path                   :",
    closure_checkpoint_path_631l,
)

print(
    "Checkpoint SHA-256                :",
    FD001_FD004_FINAL_CLOSURE_CHECKPOINT_SHA256_631L,
)

print(
    "Checkpoint roundtrip exact        :",
    closure_roundtrip_exact_631l,
)


print("\n--- PROJECT STATE ---")

print(
    "Empirical phase closed            :",
    FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L,
)

print(
    "Reporting-evidence phase closed   :",
    FD001_FD004_PROJECT_REPORTING_PHASE_CLOSED_631L,
)

print(
    "Notebook workflow closed          :",
    FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L,
)

print(
    "Ready for manuscript assembly     :",
    FD001_FD004_READY_FOR_EXTERNAL_MANUSCRIPT_ASSEMBLY_631L,
)


print(
    "\nPASS — final manuscript artifact manifest is frozen."
)

print(
    "The notebook workflow is now CLOSED. "
    "No additional scientific or reporting-evidence cells are required."
)

print(
    "Remaining work is external manuscript/thesis assembly only."
)

In [ ]:
# STEP 6.31M-A — SELECTIVE-PREDICTION PUBLICATION STRENGTHENING PRE-FLIGHT
#                    FROZEN PER-ENGINE ARTIFACT DISCOVERY
#
# PURPOSE
# -------
# Determine whether already-frozen/resident/persisted artifacts are sufficient
# for publication-strengthening analyses:
#
#   1. risk–coverage curves / AURC
#   2. matched random-rejection baseline
#   3. fixed-policy engine-level bootstrap confidence intervals
#
# DATASETS IN SCOPE
# -----------------
# FD001, FD002, FD003 only.
#
# FD004 remains structurally UNSCORED on held-out TEST and is NOT reopened.
#
# STRICT PROHIBITIONS
# -------------------
# - NO model training
# - NO model inference
# - NO MC-dropout execution
# - NO prediction regeneration
# - NO label reconstruction
# - NO raw TEST access
# - NO raw RUL-file access
# - NO threshold tuning
# - NO policy selection
# - NO new scientific metrics
# - NO modification of frozen 6.31L artifacts
#
# This step performs METADATA DISCOVERY ONLY.
#
# IMPORTANT:
# STEP 6.31L empirical/project closure remains authoritative.
# This is a publication/reporting-strengthening pre-flight only.

from pathlib import Path
import os
import re
import json
import hashlib

import numpy as np
import pandas as pd


# =============================================================================
# 0) VERIFY FINAL CLOSURE REMAINS AUTHORITATIVE
# =============================================================================

required_closure_objects_631ma = (
    "all_checks_passed_631l",
    "FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_REPORTING_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L",
    "FD001_FD004_READY_FOR_EXTERNAL_MANUSCRIPT_ASSEMBLY_631L",
    "FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L",
    "FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L",
)

missing_closure_631ma = [
    name
    for name in required_closure_objects_631ma
    if name not in globals()
]

if missing_closure_631ma:
    raise RuntimeError(
        "STEP 6.31M-A requires the completed 6.31L closure state. "
        "Missing: "
        + ", ".join(missing_closure_631ma)
    )

if not bool(all_checks_passed_631l):
    raise RuntimeError("6.31L is not in PASS state.")

if not bool(FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L):
    raise RuntimeError("Empirical phase is not closed.")

if not bool(FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L):
    raise RuntimeError("Notebook workflow is not closed.")


def _json_sha_631ma(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


if (
    _json_sha_631ma(
        FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_631L
    )
    !=
    str(
        FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L
    )
):
    raise RuntimeError(
        "6.31L final manuscript assembly manifest has changed."
    )


# =============================================================================
# 1) SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MA = False
MODEL_INFERENCE_631MA = False
MC_DROPOUT_631MA = False
RAW_TEST_ACCESS_631MA = False
RAW_RUL_ACCESS_631MA = False
LABEL_RECONSTRUCTION_631MA = False
THRESHOLD_TUNING_631MA = False
POLICY_SELECTION_631MA = False
NEW_METRICS_COMPUTED_631MA = False
REPORTING_METADATA_DISCOVERY_ONLY_631MA = True

DATASETS_IN_SCOPE_631MA = (
    "FD001",
    "FD002",
    "FD003",
)

FD004_REOPENED_631MA = False


# =============================================================================
# 2) CANDIDATE SEMANTIC KEYWORDS
#
# We want evidence that can eventually establish, per held-out engine:
#
#     y_true
#     y_pred
#     frozen risk score
#     frozen accept/abstain decision
#
# We DO NOT use those values in this step.
# =============================================================================

positive_keywords_631ma = (
    "test",
    "heldout",
    "held_out",
    "endpoint",
    "score",
    "scoring",
    "prediction",
    "pred",
    "risk",
    "accept",
    "abstain",
    "decision",
    "mask",
    "rul",
    "target",
    "truth",
    "true",
    "error",
    "evidence",
)

strong_keywords_631ma = (
    "test",
    "heldout",
    "held_out",
    "scoring",
    "prediction",
    "risk",
    "accept",
    "abstain",
)

negative_keywords_631ma = (
    "train",
    "training",
    "fit_",
    "dev_",
    "corrupt",
    "bootstrap",
    "figure",
    "caption",
    "manuscript",
)

raw_forbidden_name_patterns_631ma = (
    "test_fd001.txt",
    "test_fd002.txt",
    "test_fd003.txt",
    "rul_fd001.txt",
    "rul_fd002.txt",
    "rul_fd003.txt",
)


# =============================================================================
# 3) SAFE OBJECT METADATA SUMMARIZER
#
# No array/table values are printed.
# =============================================================================

def _object_metadata_631ma(name, obj):
    row = {
        "name": name,
        "type": type(obj).__name__,
        "shape": None,
        "dtype": None,
        "columns": None,
        "dict_keys": None,
        "length": None,
    }

    try:
        if isinstance(obj, pd.DataFrame):
            row["shape"] = tuple(obj.shape)
            row["columns"] = [str(c) for c in obj.columns]

        elif isinstance(obj, pd.Series):
            row["shape"] = tuple(obj.shape)
            row["dtype"] = str(obj.dtype)
            row["length"] = int(len(obj))

        elif isinstance(obj, np.ndarray):
            row["shape"] = tuple(obj.shape)
            row["dtype"] = str(obj.dtype)

        elif isinstance(obj, dict):
            row["length"] = int(len(obj))
            row["dict_keys"] = [
                str(k)
                for k in list(obj.keys())[:40]
            ]

        elif isinstance(obj, (list, tuple)):
            row["length"] = int(len(obj))

        else:
            if hasattr(obj, "shape"):
                try:
                    row["shape"] = tuple(obj.shape)
                except Exception:
                    pass

            if hasattr(obj, "dtype"):
                try:
                    row["dtype"] = str(obj.dtype)
                except Exception:
                    pass

    except Exception as exc:
        row["metadata_error"] = repr(exc)

    return row


# =============================================================================
# 4) RANK RESIDENT GLOBAL OBJECT CANDIDATES
# =============================================================================

def _candidate_score_631ma(name):
    lower = name.lower()

    score = 0

    for kw in positive_keywords_631ma:
        if kw in lower:
            score += 1

    for kw in strong_keywords_631ma:
        if kw in lower:
            score += 2

    for kw in negative_keywords_631ma:
        if kw in lower:
            score -= 2

    return score


resident_candidates_631ma = {
    dataset: []
    for dataset in DATASETS_IN_SCOPE_631MA
}


for name_631ma, obj_631ma in list(globals().items()):

    lower_name_631ma = name_631ma.lower()

    for dataset_631ma in DATASETS_IN_SCOPE_631MA:

        dataset_token_631ma = dataset_631ma.lower()

        if dataset_token_631ma not in lower_name_631ma:
            continue

        score_631ma = _candidate_score_631ma(
            name_631ma
        )

        if score_631ma < 2:
            continue

        metadata_631ma = _object_metadata_631ma(
            name_631ma,
            obj_631ma,
        )

        metadata_631ma["candidate_score"] = int(
            score_631ma
        )

        resident_candidates_631ma[
            dataset_631ma
        ].append(
            metadata_631ma
        )


for dataset_631ma in DATASETS_IN_SCOPE_631MA:

    resident_candidates_631ma[
        dataset_631ma
    ] = sorted(
        resident_candidates_631ma[
            dataset_631ma
        ],
        key=lambda row: (
            -row["candidate_score"],
            row["name"],
        ),
    )[:60]


# =============================================================================
# 5) DISCOVER PERSISTED RESULT FILE CANDIDATES BY FILENAME ONLY
#
# Important:
# - Walk ONLY 06_results.
# - Never walk 02_dataset/raw.
# - Never open candidate files here.
# =============================================================================

results_root_631ma = Path(
    r"C:\RUL_Reliability_Research\06_results"
)

if not results_root_631ma.is_dir():
    raise RuntimeError(
        "Expected results root not found: "
        + str(results_root_631ma)
    )


allowed_suffixes_631ma = {
    ".csv",
    ".json",
    ".npz",
    ".npy",
    ".parquet",
    ".pkl",
    ".pickle",
    ".feather",
}


persisted_candidates_631ma = {
    dataset: []
    for dataset in DATASETS_IN_SCOPE_631MA
}


for path_631ma in results_root_631ma.rglob("*"):

    if not path_631ma.is_file():
        continue

    if path_631ma.suffix.lower() not in allowed_suffixes_631ma:
        continue

    path_lower_631ma = str(
        path_631ma
    ).lower()

    basename_lower_631ma = (
        path_631ma.name.lower()
    )

    # Hard safety guard.
    if any(
        forbidden in path_lower_631ma
        for forbidden in raw_forbidden_name_patterns_631ma
    ):
        raise RuntimeError(
            "Forbidden raw TEST/RUL path unexpectedly "
            "encountered inside result scan: "
            + str(path_631ma)
        )

    for dataset_631ma in DATASETS_IN_SCOPE_631MA:

        dataset_token_631ma = (
            dataset_631ma.lower()
        )

        if dataset_token_631ma not in path_lower_631ma:
            continue

        score_631ma = _candidate_score_631ma(
            basename_lower_631ma
        )

        # Also score parent-directory semantics.
        score_631ma += max(
            0,
            _candidate_score_631ma(
                path_lower_631ma
            ) // 3,
        )

        if score_631ma < 2:
            continue

        persisted_candidates_631ma[
            dataset_631ma
        ].append(
            {
                "path": str(path_631ma),
                "suffix": path_631ma.suffix.lower(),
                "size_bytes": int(
                    path_631ma.stat().st_size
                ),
                "candidate_score": int(
                    score_631ma
                ),
            }
        )


for dataset_631ma in DATASETS_IN_SCOPE_631MA:

    persisted_candidates_631ma[
        dataset_631ma
    ] = sorted(
        persisted_candidates_631ma[
            dataset_631ma
        ],
        key=lambda row: (
            -row["candidate_score"],
            row["path"],
        ),
    )[:80]


# =============================================================================
# 6) SEMANTIC COVERAGE HEURISTIC
#
# This does NOT certify authority or scientific usability.
# It merely tells us whether names/metadata suggest that each required
# per-engine ingredient may already exist.
# =============================================================================

ingredient_terms_631ma = {
    "prediction": (
        "pred",
        "prediction",
    ),
    "truth_or_target": (
        "truth",
        "true",
        "target",
        "rul",
        "label",
    ),
    "risk": (
        "risk",
        "uncertainty",
    ),
    "decision": (
        "accept",
        "abstain",
        "decision",
        "mask",
    ),
}


ingredient_presence_631ma = {}


for dataset_631ma in DATASETS_IN_SCOPE_631MA:

    searchable_strings_631ma = []

    searchable_strings_631ma.extend(
        row["name"].lower()
        for row in resident_candidates_631ma[
            dataset_631ma
        ]
    )

    searchable_strings_631ma.extend(
        row["path"].lower()
        for row in persisted_candidates_631ma[
            dataset_631ma
        ]
    )

    combined_631ma = "\n".join(
        searchable_strings_631ma
    )

    ingredient_presence_631ma[
        dataset_631ma
    ] = {
        ingredient: bool(
            any(
                term in combined_631ma
                for term in terms
            )
        )
        for ingredient, terms
        in ingredient_terms_631ma.items()
    }


# =============================================================================
# 7) FREEZE THE PRE-FLIGHT AUDIT OBJECT
# =============================================================================

FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_631MA = {
    "stage": "6.31M-A",
    "purpose":
        "metadata-only frozen per-engine artifact discovery",

    "closure_manifest_sha256":
        str(
            FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L
        ),

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "new_metrics_computed":
        False,

    "fd004_reopened":
        False,

    "datasets":
        list(
            DATASETS_IN_SCOPE_631MA
        ),

    "ingredient_presence_heuristic":
        ingredient_presence_631ma,

    "resident_candidate_counts": {
        dataset: len(
            resident_candidates_631ma[
                dataset
            ]
        )
        for dataset in DATASETS_IN_SCOPE_631MA
    },

    "persisted_candidate_counts": {
        dataset: len(
            persisted_candidates_631ma[
                dataset
            ]
        )
        for dataset in DATASETS_IN_SCOPE_631MA
    },
}


FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_SHA256_631MA = (
    _json_sha_631ma(
        FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_631MA
    )
)


all_checks_passed_631ma = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-A — SELECTIVE-PREDICTION PUBLICATION "
    "STRENGTHENING PRE-FLIGHT"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("New scientific metrics            :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


for dataset_631ma in DATASETS_IN_SCOPE_631MA:

    print(
        "\n"
        + "-" * 176
    )

    print(
        dataset_631ma,
        "— REQUIRED INGREDIENT HEURISTIC"
    )

    print(
        "-" * 176
    )

    for ingredient_631ma, present_631ma in (
        ingredient_presence_631ma[
            dataset_631ma
        ].items()
    ):
        print(
            f"{ingredient_631ma:30s}:",
            present_631ma,
        )

    print(
        "\nResident candidate objects:",
        len(
            resident_candidates_631ma[
                dataset_631ma
            ]
        ),
    )

    for row_631ma in (
        resident_candidates_631ma[
            dataset_631ma
        ][:25]
    ):

        print(
            "  ",
            f"[score={row_631ma['candidate_score']:>2}]",
            row_631ma["name"],
            "| type=",
            row_631ma["type"],
            "| shape=",
            row_631ma["shape"],
            "| columns=",
            row_631ma["columns"],
            "| keys=",
            row_631ma["dict_keys"],
        )

    print(
        "\nPersisted result-file candidates:",
        len(
            persisted_candidates_631ma[
                dataset_631ma
            ]
        ),
    )

    for row_631ma in (
        persisted_candidates_631ma[
            dataset_631ma
        ][:30]
    ):

        print(
            "  ",
            f"[score={row_631ma['candidate_score']:>2}]",
            row_631ma["path"],
            "|",
            row_631ma["size_bytes"],
            "bytes",
        )


print(
    "\n"
    + "-" * 176
)

print(
    "Pre-flight contract SHA-256       :",
    FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_SHA256_631MA,
)

print(
    "All pre-flight checks passed      :",
    all_checks_passed_631ma,
)

print(
    "\nPASS — publication-strengthening pre-flight completed "
    "without reopening the empirical pipeline."
)

print(
    "No risk–coverage, random-rejection, bootstrap, "
    "or other publication metric has been computed yet."
)

In [ ]:
# STEP 6.31M-B — AUTHORITATIVE PUBLICATION-ANALYSIS SCHEMA +
#                 FD003 HASH-IDENTITY RECOVERY
#
# PURPOSE
# -------
# Resolve which already-frozen artifacts/objects can legitimately support:
#   - risk–coverage analysis,
#   - matched random-rejection analysis,
#   - fixed-policy bootstrap reporting.
#
# This step DOES NOT compute any of those analyses.
#
# It performs:
#   1) schema/key inspection of frozen FD001/FD002 result artifacts;
#   2) provenance-field inspection of their metadata;
#   3) filename-only search for FD003 persisted evidence;
#   4) resident-object SHA identity matching against the frozen
#      FD003 TEST contracts.
#
# STRICTLY REPORTING/PROVENANCE ONLY.
#
# NO:
#   - model inference
#   - MC dropout
#   - prediction regeneration
#   - label reconstruction
#   - raw TEST/RUL access
#   - threshold/policy change
#   - risk–coverage/AURC calculation
#   - random-rejection calculation
#   - bootstrap calculation
#   - new scientific metric

from pathlib import Path
import io
import json
import hashlib

import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631mb = (
    "all_checks_passed_631l",
    "all_checks_passed_631ma",
    "FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L",
    "FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_SHA256_631MA",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
)

missing_631mb = [
    name for name in required_631mb
    if name not in globals()
]

if missing_631mb:
    raise RuntimeError(
        "STEP 6.31M-B missing required frozen state: "
        + ", ".join(missing_631mb)
    )

if not bool(all_checks_passed_631l):
    raise RuntimeError("6.31L closure is not in PASS state.")

if not bool(all_checks_passed_631ma):
    raise RuntimeError("6.31M-A pre-flight is not in PASS state.")

if not bool(FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L):
    raise RuntimeError("Empirical phase is not closed.")

if not bool(FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L):
    raise RuntimeError("Notebook workflow closure is not preserved.")


# =============================================================================
# 1) SAFETY FLAGS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MB = False
RAW_TEST_ACCESS_631MB = False
RAW_RUL_ACCESS_631MB = False
MODEL_INFERENCE_631MB = False
MC_DROPOUT_631MB = False
PREDICTION_REGENERATION_631MB = False
LABEL_RECONSTRUCTION_631MB = False
THRESHOLD_TUNING_631MB = False
POLICY_SELECTION_631MB = False
RISK_COVERAGE_COMPUTED_631MB = False
RANDOM_REJECTION_COMPUTED_631MB = False
BOOTSTRAP_COMPUTED_631MB = False
NEW_SCIENTIFIC_METRIC_631MB = False
FD004_REOPENED_631MB = False


# =============================================================================
# 2) HELPERS — METADATA / SCHEMA ONLY
# =============================================================================

def _json_sha_631mb(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def _csv_schema_631mb(path):
    path = Path(path)

    if not path.is_file():
        return {
            "exists": False,
            "path": str(path),
        }

    df0 = pd.read_csv(path, nrows=0)

    return {
        "exists": True,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "columns": [str(c) for c in df0.columns],
    }


def _npz_schema_631mb(path):
    path = Path(path)

    if not path.is_file():
        return {
            "exists": False,
            "path": str(path),
        }

    out = {
        "exists": True,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "keys": {},
    }

    with np.load(path, allow_pickle=False) as z:
        for key in z.files:
            arr = z[key]
            out["keys"][str(key)] = {
                "shape": tuple(arr.shape),
                "dtype": str(arr.dtype),
            }

    return out


def _json_metadata_631mb(path, wanted_keys=None):
    path = Path(path)

    if not path.is_file():
        return {
            "exists": False,
            "path": str(path),
        }

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    out = {
        "exists": True,
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "top_level_keys": (
            list(obj.keys())
            if isinstance(obj, dict)
            else None
        ),
    }

    if isinstance(obj, dict) and wanted_keys:
        out["selected_fields"] = {
            k: obj.get(k)
            for k in wanted_keys
            if k in obj
        }

    return out


# =============================================================================
# 3) FD001 — INSPECT FROZEN RESULT SCHEMAS
#
# IMPORTANT:
# The existence of old corruption-transfer material does NOT automatically
# make it authoritative for the final 6.31I target-aware result.
# This step only exposes schema/provenance for later identity adjudication.
# =============================================================================

fd001_clean_reference_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_fd001\corruption_transfer\full_scoring"
    r"\fd001_official_test_clean_endpoint_scoring_reference.csv"
)

fd001_prescore_npz_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_test"
    r"\fd001_official_endpoint_prescore_arrays.npz"
)

fd001_prescore_csv_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_test"
    r"\fd001_official_endpoint_prescore_inference.csv"
)

fd001_risk_curve_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_fd001\corruption_transfer\full_scoring\risk_coverage"
    r"\fd001_risk_coverage_clean_curve.csv"
)

fd001_risk_manifest_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_fd001\corruption_transfer\full_scoring\risk_coverage"
    r"\fd001_risk_coverage_manifest.json"
)

fd001_clean_summary_path_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_official_fd001"
    r"\fd001_official_clean_scoring_summary.json"
)


FD001_PUBLICATION_SCHEMA_631MB = {
    "clean_endpoint_scoring_reference":
        _csv_schema_631mb(
            fd001_clean_reference_path_631mb
        ),

    "prescore_arrays":
        _npz_schema_631mb(
            fd001_prescore_npz_path_631mb
        ),

    "prescore_inference":
        _csv_schema_631mb(
            fd001_prescore_csv_path_631mb
        ),

    "existing_risk_coverage_clean_curve":
        _csv_schema_631mb(
            fd001_risk_curve_path_631mb
        ),

    "risk_coverage_manifest":
        _json_metadata_631mb(
            fd001_risk_manifest_path_631mb,
            wanted_keys=(
                "dataset",
                "step",
                "status",
                "target",
                "evaluation_target",
                "threshold",
                "risk_threshold",
                "coverage",
                "source",
                "source_sha256",
                "scoring_reference_sha256",
            ),
        ),

    "clean_scoring_summary":
        _json_metadata_631mb(
            fd001_clean_summary_path_631mb,
            wanted_keys=(
                "dataset",
                "step",
                "status",
                "target",
                "evaluation_target",
                "coverage",
                "threshold",
                "full_mae",
                "accepted_mae",
                "abstained_mae",
            ),
        ),
}


# =============================================================================
# 4) FD002 — INSPECT AUTHORITATIVE C5 SCHEMAS
# =============================================================================

fd002_root_631mb = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_reliability_signals\fd002_c5"
)

fd002_prediction_csv_631mb = (
    fd002_root_631mb
    / "fd002_c5_canonical_test_prediction.csv"
)

fd002_prediction_npz_631mb = (
    fd002_root_631mb
    / "fd002_c5_canonical_test_prediction.npz"
)

fd002_prediction_meta_631mb = (
    fd002_root_631mb
    / "fd002_c5_canonical_test_prediction_metadata.json"
)

fd002_scoring_csv_631mb = (
    fd002_root_631mb
    / "fd002_c5_official_test_scoring.csv"
)

fd002_scoring_npz_631mb = (
    fd002_root_631mb
    / "fd002_c5_official_test_scoring.npz"
)

fd002_scoring_meta_631mb = (
    fd002_root_631mb
    / "fd002_c5_official_test_scoring_metadata.json"
)

fd002_decisions_csv_631mb = (
    fd002_root_631mb
    / "fd002_c5_test_operating_decisions.csv"
)

fd002_decisions_npz_631mb = (
    fd002_root_631mb
    / "fd002_c5_test_reliability_and_decisions.npz"
)

fd002_decisions_meta_631mb = (
    fd002_root_631mb
    / "fd002_c5_test_operating_decisions_metadata.json"
)


FD002_PUBLICATION_SCHEMA_631MB = {
    "canonical_prediction_csv":
        _csv_schema_631mb(
            fd002_prediction_csv_631mb
        ),

    "canonical_prediction_npz":
        _npz_schema_631mb(
            fd002_prediction_npz_631mb
        ),

    "canonical_prediction_metadata":
        _json_metadata_631mb(
            fd002_prediction_meta_631mb,
            wanted_keys=(
                "dataset",
                "step",
                "status",
                "model_retrained",
                "model_executed",
                "prediction_sha256",
                "test_endpoints",
            ),
        ),

    "official_scoring_csv":
        _csv_schema_631mb(
            fd002_scoring_csv_631mb
        ),

    "official_scoring_npz":
        _npz_schema_631mb(
            fd002_scoring_npz_631mb
        ),

    "official_scoring_metadata":
        _json_metadata_631mb(
            fd002_scoring_meta_631mb,
            wanted_keys=(
                "dataset",
                "step",
                "status",
                "evaluation_target",
                "rul_cap",
                "test_endpoints",
                "accepted_test_endpoints",
                "abstained_test_endpoints",
                "realized_test_coverage",
                "frozen_exact_risk_threshold",
                "threshold_tuned_after_test",
                "threshold_modified_after_test",
                "model_retrained_after_test",
            ),
        ),

    "operating_decisions_csv":
        _csv_schema_631mb(
            fd002_decisions_csv_631mb
        ),

    "reliability_decisions_npz":
        _npz_schema_631mb(
            fd002_decisions_npz_631mb
        ),

    "operating_decisions_metadata":
        _json_metadata_631mb(
            fd002_decisions_meta_631mb,
            wanted_keys=(
                "dataset",
                "step",
                "status",
                "decision_rule",
                "frozen_risk_threshold",
                "nominal_coverage",
                "official_test_labels_read",
                "accepted_count",
                "abstained_count",
            ),
        ),
}


# =============================================================================
# 5) FD003 — SEARCH ONLY SAFE RESULT/CHECKPOINT LOCATIONS BY FILENAME
# =============================================================================

safe_fd003_roots_631mb = (
    Path(r"C:\RUL_Reliability_Research\06_results"),
    Path(r"C:\RUL_Reliability_Research\checkpoints"),
)

fd003_filename_tokens_631mb = (
    "fd003",
    "629ba",
    "629bb",
    "heldout",
    "test_score",
    "test_result",
)

fd003_file_candidates_631mb = []

allowed_suffixes_631mb = {
    ".csv",
    ".json",
    ".npz",
    ".npy",
    ".parquet",
    ".pkl",
    ".pickle",
    ".feather",
}

for root_631mb in safe_fd003_roots_631mb:

    if not root_631mb.is_dir():
        continue

    for path_631mb in root_631mb.rglob("*"):

        if not path_631mb.is_file():
            continue

        if (
            path_631mb.suffix.lower()
            not in allowed_suffixes_631mb
        ):
            continue

        low_631mb = str(path_631mb).lower()

        if not any(
            token in low_631mb
            for token in fd003_filename_tokens_631mb
        ):
            continue

        fd003_file_candidates_631mb.append(
            {
                "path": str(path_631mb),
                "suffix": path_631mb.suffix.lower(),
                "size_bytes": int(
                    path_631mb.stat().st_size
                ),
            }
        )

fd003_file_candidates_631mb = sorted(
    fd003_file_candidates_631mb,
    key=lambda x: x["path"],
)


# =============================================================================
# 6) FD003 — EXPECTED FROZEN ARRAY HASHES
# =============================================================================

fd003_feature_contract_631mb = (
    FD003_TEST_FEATURE_SIDE_CONTRACT_629BA
)

fd003_score_contract_631mb = (
    FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB
)

expected_fd003_hashes_631mb = {}

for key_631mb in (
    "deterministic_prediction_sha256",
    "final_selective_risk_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
    "g1_risk_sha256",
    "phase_adjusted_uncertainty_sha256",
    "mc_mean_sha256",
    "mc_std_sha256",
    "endpoint_engine_ids_sha256",
    "endpoint_cycles_sha256",
):

    if key_631mb in fd003_feature_contract_631mb:
        expected_fd003_hashes_631mb[
            key_631mb
        ] = str(
            fd003_feature_contract_631mb[
                key_631mb
            ]
        )


for key_631mb in (
    "absolute_error_sha256",
    "endpoint_results_sha256",
):

    if key_631mb in fd003_score_contract_631mb:
        expected_fd003_hashes_631mb[
            key_631mb
        ] = str(
            fd003_score_contract_631mb[
                key_631mb
            ]
        )


# =============================================================================
# 7) HASH VARIANTS FOR RESIDENT ARRAY IDENTITY
#
# We do NOT assume which historical serialization convention was used.
# Several harmless deterministic variants are checked.
# Values are NEVER printed.
# =============================================================================

def _resident_array_631mb(obj):

    if isinstance(obj, np.ndarray):
        return np.asarray(obj)

    if isinstance(obj, pd.Series):
        return obj.to_numpy()

    if isinstance(obj, pd.Index):
        return obj.to_numpy()

    if isinstance(obj, (list, tuple)):
        try:
            arr = np.asarray(obj)

            if arr.dtype != object:
                return arr
        except Exception:
            return None

    return None


def _hash_variants_631mb(arr):

    arr = np.asarray(arr)

    if arr.dtype == object:
        return {}

    arr_c = np.ascontiguousarray(arr)

    out = {}

    # Raw contiguous bytes.
    out["raw_c_bytes"] = hashlib.sha256(
        arr_c.tobytes(order="C")
    ).hexdigest()

    # Shape + dtype + bytes.
    prefix = (
        str(arr_c.shape)
        + "|"
        + str(arr_c.dtype)
        + "|"
    ).encode("utf-8")

    out["shape_dtype_raw"] = hashlib.sha256(
        prefix
        + arr_c.tobytes(order="C")
    ).hexdigest()

    # NPY serialization.
    try:
        bio = io.BytesIO()

        np.save(
            bio,
            arr_c,
            allow_pickle=False,
        )

        out["npy_serialized"] = hashlib.sha256(
            bio.getvalue()
        ).hexdigest()

    except Exception:
        pass

    return out


# =============================================================================
# 8) SEARCH RESIDENT OBJECTS FOR FD003 HASH MATCHES
#
# Endpoint count is frozen at 100.
# We examine only compact resident arrays likely to represent endpoint-level
# TEST evidence. No values are displayed.
# =============================================================================

fd003_endpoint_n_631mb = int(
    fd003_score_contract_631mb[
        "endpoint_count"
    ]
)

fd003_resident_candidates_631mb = []
fd003_exact_hash_matches_631mb = []


for name_631mb, obj_631mb in list(globals().items()):

    if name_631mb.startswith("_"):
        continue

    arr_631mb = _resident_array_631mb(
        obj_631mb
    )

    if arr_631mb is None:
        continue

    # Endpoint-level arrays only.
    if (
        arr_631mb.size
        != fd003_endpoint_n_631mb
    ):
        continue

    hashes_631mb = _hash_variants_631mb(
        arr_631mb
    )

    metadata_631mb = {
        "name": name_631mb,
        "shape": tuple(arr_631mb.shape),
        "dtype": str(arr_631mb.dtype),
        "hash_variants": hashes_631mb,
    }

    fd003_resident_candidates_631mb.append(
        metadata_631mb
    )

    for semantic_key_631mb, expected_hash_631mb in (
        expected_fd003_hashes_631mb.items()
    ):

        for method_631mb, actual_hash_631mb in (
            hashes_631mb.items()
        ):

            if actual_hash_631mb == expected_hash_631mb:

                fd003_exact_hash_matches_631mb.append(
                    {
                        "semantic_key":
                            semantic_key_631mb,

                        "resident_object":
                            name_631mb,

                        "hash_method":
                            method_631mb,

                        "sha256":
                            actual_hash_631mb,

                        "shape":
                            tuple(arr_631mb.shape),

                        "dtype":
                            str(arr_631mb.dtype),
                    }
                )


fd003_resident_candidates_631mb = sorted(
    fd003_resident_candidates_631mb,
    key=lambda x: x["name"],
)

fd003_exact_hash_matches_631mb = sorted(
    fd003_exact_hash_matches_631mb,
    key=lambda x: (
        x["semantic_key"],
        x["resident_object"],
        x["hash_method"],
    ),
)


# =============================================================================
# 9) ALSO REPORT FD003 RESIDENT SEMANTIC NAMES
#
# These are metadata clues only, not authority claims.
# =============================================================================

fd003_semantic_name_tokens_631mb = (
    "fd003",
    "629ba",
    "629bb",
    "test",
    "heldout",
    "pred",
    "risk",
    "accept",
    "abstain",
    "error",
    "rul",
)

fd003_semantic_resident_names_631mb = []

for row_631mb in fd003_resident_candidates_631mb:

    low_name_631mb = row_631mb[
        "name"
    ].lower()

    if any(
        token in low_name_631mb
        for token in fd003_semantic_name_tokens_631mb
    ):
        fd003_semantic_resident_names_631mb.append(
            {
                "name": row_631mb["name"],
                "shape": row_631mb["shape"],
                "dtype": row_631mb["dtype"],
            }
        )


# =============================================================================
# 10) FREEZE AUDIT OBJECT
# =============================================================================

FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_631MB = {
    "stage": "6.31M-B",

    "preflight_sha256":
        str(
            FD001_FD003_PUBLICATION_STRENGTHENING_PREFLIGHT_SHA256_631MA
        ),

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "risk_coverage_computed":
        False,

    "random_rejection_computed":
        False,

    "bootstrap_computed":
        False,

    "fd004_reopened":
        False,

    "fd001_schema":
        FD001_PUBLICATION_SCHEMA_631MB,

    "fd002_schema":
        FD002_PUBLICATION_SCHEMA_631MB,

    "fd003_expected_hashes":
        expected_fd003_hashes_631mb,

    "fd003_safe_file_candidates":
        fd003_file_candidates_631mb,

    "fd003_exact_hash_matches":
        fd003_exact_hash_matches_631mb,

    "fd003_semantic_resident_names":
        fd003_semantic_resident_names_631mb,
}


FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_SHA256_631MB = (
    _json_sha_631mb(
        FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_631MB
    )
)


all_checks_passed_631mb = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-B — AUTHORITATIVE PUBLICATION-ANALYSIS "
    "SCHEMA + FD003 HASH-IDENTITY RECOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Risk–coverage computed            :", False)
print("Random rejection computed         :", False)
print("Bootstrap computed                :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


print("\n--- FD001 FROZEN ARTIFACT SCHEMA ---")

for key_631mb, value_631mb in (
    FD001_PUBLICATION_SCHEMA_631MB.items()
):
    print("\n", key_631mb)
    print(json.dumps(
        value_631mb,
        indent=2,
        default=str,
    ))


print("\n--- FD002 FROZEN C5 ARTIFACT SCHEMA ---")

for key_631mb, value_631mb in (
    FD002_PUBLICATION_SCHEMA_631MB.items()
):
    print("\n", key_631mb)
    print(json.dumps(
        value_631mb,
        indent=2,
        default=str,
    ))


print("\n--- FD003 SAFE PERSISTED FILE CANDIDATES ---")

print(
    "Candidate count:",
    len(
        fd003_file_candidates_631mb
    ),
)

for row_631mb in (
    fd003_file_candidates_631mb[:100]
):
    print(
        " ",
        row_631mb["path"],
        "|",
        row_631mb["size_bytes"],
        "bytes",
    )


print("\n--- FD003 EXPECTED FROZEN HASHES ---")

for key_631mb, value_631mb in (
    expected_fd003_hashes_631mb.items()
):
    print(
        f"{key_631mb:40s}:",
        value_631mb,
    )


print("\n--- FD003 EXACT RESIDENT HASH MATCHES ---")

print(
    "Exact match count:",
    len(
        fd003_exact_hash_matches_631mb
    ),
)

for row_631mb in (
    fd003_exact_hash_matches_631mb
):
    print(
        " ",
        row_631mb["semantic_key"],
        "->",
        row_631mb["resident_object"],
        "| method=",
        row_631mb["hash_method"],
        "| shape=",
        row_631mb["shape"],
        "| dtype=",
        row_631mb["dtype"],
    )


print("\n--- FD003 SEMANTIC RESIDENT ENDPOINT OBJECTS ---")

print(
    "Candidate count:",
    len(
        fd003_semantic_resident_names_631mb
    ),
)

for row_631mb in (
    fd003_semantic_resident_names_631mb[:100]
):
    print(
        " ",
        row_631mb["name"],
        "| shape=",
        row_631mb["shape"],
        "| dtype=",
        row_631mb["dtype"],
    )


print(
    "\nPublication authority audit SHA-256:",
    FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_SHA256_631MB,
)

print(
    "All checks passed                  :",
    all_checks_passed_631mb,
)

print(
    "\nPASS — schema/provenance inspection completed. "
    "No publication-strengthening metric has been calculated yet."
)

In [ ]:
# STEP 6.31M-C — FD001 FINAL-LINEAGE + FD003 CHECKPOINT PAYLOAD RECOVERY
#
# PURPOSE
# -------
# Resolve the remaining publication-analysis provenance gaps WITHOUT
# recomputing science.
#
# FD001:
#   - inspect ONLY the final frozen fd001_final result directory;
#   - identify whether final 6.31I-compatible row-level prediction/risk/
#     decision/target evidence already exists;
#   - explicitly keep older STEP 6.25F risk-coverage material separate.
#
# FD003:
#   - inspect the frozen restart checkpoint;
#   - recursively locate embedded endpoint-level arrays/records;
#   - match them cryptographically against frozen 629BA/629BB hashes.
#
# STRICTLY PROHIBITED
# -------------------
# - raw TEST/RUL access
# - model inference
# - MC dropout
# - prediction regeneration
# - label reconstruction
# - threshold/policy modification
# - AURC/risk-coverage calculation
# - random rejection
# - bootstrap
# - any new scientific metric

from pathlib import Path
import io
import json
import hashlib
import re

import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631mc = (
    "all_checks_passed_631l",
    "all_checks_passed_631ma",
    "all_checks_passed_631mb",
    "FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L",
    "FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_SHA256_631MB",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
)

missing_631mc = [
    name for name in required_631mc
    if name not in globals()
]

if missing_631mc:
    raise RuntimeError(
        "STEP 6.31M-C missing required frozen state: "
        + ", ".join(missing_631mc)
    )

if not bool(all_checks_passed_631l):
    raise RuntimeError("6.31L is not in PASS state.")

if not bool(all_checks_passed_631ma):
    raise RuntimeError("6.31M-A is not in PASS state.")

if not bool(all_checks_passed_631mb):
    raise RuntimeError("6.31M-B is not in PASS state.")

if not bool(FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L):
    raise RuntimeError("Empirical closure is not preserved.")

if not bool(FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L):
    raise RuntimeError("Notebook closure is not preserved.")


# =============================================================================
# 1) SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MC = False
RAW_TEST_ACCESS_631MC = False
RAW_RUL_ACCESS_631MC = False
MODEL_INFERENCE_631MC = False
MC_DROPOUT_631MC = False
PREDICTION_REGENERATION_631MC = False
LABEL_RECONSTRUCTION_631MC = False
THRESHOLD_TUNING_631MC = False
POLICY_SELECTION_631MC = False
RISK_COVERAGE_COMPUTED_631MC = False
AURC_COMPUTED_631MC = False
RANDOM_REJECTION_COMPUTED_631MC = False
BOOTSTRAP_COMPUTED_631MC = False
NEW_SCIENTIFIC_METRIC_631MC = False
FD004_REOPENED_631MC = False


# =============================================================================
# 2) HELPERS
# =============================================================================

def _json_sha_631mc(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def _file_sha_631mc(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def _array_hash_variants_631mc(arr):
    """
    Harmless deterministic SHA variants only.
    No values are printed.
    """
    arr = np.asarray(arr)

    if arr.dtype == object:
        return {}

    arr_c = np.ascontiguousarray(arr)

    variants = {}

    variants["raw_c_bytes"] = hashlib.sha256(
        arr_c.tobytes(order="C")
    ).hexdigest()

    prefix = (
        str(arr_c.shape)
        + "|"
        + str(arr_c.dtype)
        + "|"
    ).encode("utf-8")

    variants["shape_dtype_raw"] = hashlib.sha256(
        prefix + arr_c.tobytes(order="C")
    ).hexdigest()

    try:
        bio = io.BytesIO()

        np.save(
            bio,
            arr_c,
            allow_pickle=False,
        )

        variants["npy_serialized"] = hashlib.sha256(
            bio.getvalue()
        ).hexdigest()

    except Exception:
        pass

    # Useful numeric canonicalizations sometimes used historically.
    if arr_c.dtype.kind in "biuf":
        for dtype_name in (
            "float64",
            "float32",
            "int64",
            "int32",
        ):
            try:
                canon = np.ascontiguousarray(
                    arr_c.astype(
                        dtype_name,
                        copy=False,
                    )
                )

                variants[
                    f"raw_as_{dtype_name}"
                ] = hashlib.sha256(
                    canon.tobytes(order="C")
                ).hexdigest()

            except Exception:
                pass

    if arr_c.dtype.kind == "b":
        variants["raw_as_bool"] = hashlib.sha256(
            np.ascontiguousarray(
                arr_c.astype(bool)
            ).tobytes(order="C")
        ).hexdigest()

    return variants


def _safe_schema_631mc(path):
    """
    Schema/provenance only.
    """
    path = Path(path)

    out = {
        "path": str(path),
        "exists": path.is_file(),
    }

    if not path.is_file():
        return out

    out["size_bytes"] = int(path.stat().st_size)
    out["file_sha256"] = _file_sha_631mc(path)
    out["suffix"] = path.suffix.lower()

    suffix = path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(path, nrows=0)

        out["columns"] = [
            str(c)
            for c in header.columns
        ]

    elif suffix == ".npz":
        with np.load(
            path,
            allow_pickle=False,
        ) as z:
            out["keys"] = {
                str(key): {
                    "shape": tuple(
                        z[key].shape
                    ),
                    "dtype": str(
                        z[key].dtype
                    ),
                }
                for key in z.files
            }

    elif suffix == ".npy":
        arr = np.load(
            path,
            allow_pickle=False,
            mmap_mode="r",
        )

        out["shape"] = tuple(arr.shape)
        out["dtype"] = str(arr.dtype)

    elif suffix == ".json":
        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            out["top_level_keys"] = [
                str(k)
                for k in obj.keys()
            ]

    return out


# =============================================================================
# 3) FD001 — FINAL DIRECTORY ONLY
# =============================================================================

fd001_final_root_631mc = Path(
    r"C:\RUL_Reliability_Research\06_results\fd001_final"
)

if not fd001_final_root_631mc.is_dir():
    raise RuntimeError(
        "Expected frozen FD001 final directory not found: "
        + str(fd001_final_root_631mc)
    )


allowed_suffixes_631mc = {
    ".csv",
    ".json",
    ".npz",
    ".npy",
    ".parquet",
    ".feather",
    ".pkl",
    ".pickle",
}


fd001_final_files_631mc = []

for p in fd001_final_root_631mc.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in allowed_suffixes_631mc:
        continue

    fd001_final_files_631mc.append(
        _safe_schema_631mc(p)
    )


fd001_final_files_631mc = sorted(
    fd001_final_files_631mc,
    key=lambda x: x["path"],
)


# =============================================================================
# 4) FD001 — READ FINAL EVIDENCE JSON METADATA/PROVENANCE
# =============================================================================

fd001_final_evidence_path_631mc = (
    fd001_final_root_631mc
    / "fd001_final_evidence.json"
)

if not fd001_final_evidence_path_631mc.is_file():
    raise RuntimeError(
        "Authoritative FD001 final evidence JSON not found: "
        + str(fd001_final_evidence_path_631mc)
    )


with open(
    fd001_final_evidence_path_631mc,
    "r",
    encoding="utf-8",
) as f:
    fd001_final_evidence_631mc = json.load(f)


def _walk_json_metadata_631mc(
    obj,
    prefix="root",
    depth=0,
    max_depth=8,
):
    """
    Return paths/types only plus scalar values for provenance-relevant fields.
    Does not print arbitrary array contents.
    """
    rows = []

    if depth > max_depth:
        return rows

    if isinstance(obj, dict):

        for key, value in obj.items():

            path = f"{prefix}.{key}"

            key_low = str(key).lower()

            scalar_allowed = any(
                token in key_low
                for token in (
                    "sha",
                    "path",
                    "file",
                    "source",
                    "target",
                    "threshold",
                    "coverage",
                    "status",
                    "step",
                    "count",
                    "endpoint",
                    "accepted",
                    "abstained",
                    "policy",
                    "risk",
                    "prediction",
                    "score",
                    "rul",
                )
            )

            row = {
                "path": path,
                "type": type(value).__name__,
            }

            if (
                scalar_allowed
                and isinstance(
                    value,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                )
            ):
                row["value"] = value

            elif isinstance(
                value,
                (list, tuple),
            ):
                row["length"] = len(value)

            elif isinstance(value, dict):
                row["keys"] = [
                    str(k)
                    for k in value.keys()
                ]

            rows.append(row)

            rows.extend(
                _walk_json_metadata_631mc(
                    value,
                    prefix=path,
                    depth=depth + 1,
                    max_depth=max_depth,
                )
            )

    elif isinstance(obj, list):

        rows.append(
            {
                "path": prefix,
                "type": "list",
                "length": len(obj),
            }
        )

    return rows


fd001_final_evidence_metadata_631mc = (
    _walk_json_metadata_631mc(
        fd001_final_evidence_631mc
    )
)


# =============================================================================
# 5) FD001 — FLAG ROW-LEVEL CANDIDATE FILES
#
# No values are used.
# =============================================================================

fd001_rowlevel_tokens_631mc = (
    "pred",
    "prediction",
    "risk",
    "reliab",
    "accept",
    "abstain",
    "decision",
    "score",
    "endpoint",
    "rul",
    "error",
)


fd001_rowlevel_candidates_631mc = []

for row in fd001_final_files_631mc:

    searchable = (
        row["path"].lower()
        + " "
        + " ".join(
            str(c).lower()
            for c in row.get(
                "columns",
                [],
            )
        )
        + " "
        + " ".join(
            str(k).lower()
            for k in row.get(
                "keys",
                {},
            )
        )
    )

    hits = [
        token
        for token in fd001_rowlevel_tokens_631mc
        if token in searchable
    ]

    if hits:
        candidate = dict(row)
        candidate["semantic_hits"] = hits

        fd001_rowlevel_candidates_631mc.append(
            candidate
        )


# =============================================================================
# 6) FD003 — LOAD FROZEN RESTART CHECKPOINT
# =============================================================================

fd003_checkpoint_path_631mc = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd003_post_629bc_restart_checkpoint_v1.json"
)

if not fd003_checkpoint_path_631mc.is_file():
    raise RuntimeError(
        "FD003 frozen restart checkpoint not found: "
        + str(fd003_checkpoint_path_631mc)
    )


with open(
    fd003_checkpoint_path_631mc,
    "r",
    encoding="utf-8",
) as f:
    fd003_checkpoint_631mc = json.load(f)


fd003_checkpoint_sha256_631mc = (
    _file_sha_631mc(
        fd003_checkpoint_path_631mc
    )
)


# =============================================================================
# 7) FD003 EXPECTED HASH AUTHORITY
# =============================================================================

fd003_expected_hashes_631mc = {}

for key in (
    "deterministic_prediction_sha256",
    "final_selective_risk_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
    "g1_risk_sha256",
    "phase_adjusted_uncertainty_sha256",
    "mc_mean_sha256",
    "mc_std_sha256",
    "endpoint_engine_ids_sha256",
    "endpoint_cycles_sha256",
):

    if key in FD003_TEST_FEATURE_SIDE_CONTRACT_629BA:

        fd003_expected_hashes_631mc[
            key
        ] = str(
            FD003_TEST_FEATURE_SIDE_CONTRACT_629BA[
                key
            ]
        )


for key in (
    "absolute_error_sha256",
    "endpoint_results_sha256",
):

    if key in FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB:

        fd003_expected_hashes_631mc[
            key
        ] = str(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                key
            ]
        )


# =============================================================================
# 8) FD003 — RECURSIVELY EXTRACT ENDPOINT-SIZED NUMERIC ARRAYS
#
# Accepted structures:
#   - list/tuple of 100 scalar numeric/bool values
#   - dict values containing such lists
#   - list of 100 records -> each scalar column is recovered
#
# Values are never printed.
# =============================================================================

FD003_ENDPOINT_N_631MC = int(
    FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
        "endpoint_count"
    ]
)


fd003_checkpoint_arrays_631mc = []


def _record_array_candidate_631mc(
    path,
    arr,
    origin,
):
    arr = np.asarray(arr)

    if arr.size != FD003_ENDPOINT_N_631MC:
        return

    if arr.dtype == object:
        return

    fd003_checkpoint_arrays_631mc.append(
        {
            "path": path,
            "origin": origin,
            "shape": tuple(arr.shape),
            "dtype": str(arr.dtype),
            "hash_variants":
                _array_hash_variants_631mc(
                    arr
                ),
        }
    )


def _walk_checkpoint_631mc(
    obj,
    path="root",
    depth=0,
    max_depth=12,
):

    if depth > max_depth:
        return

    if isinstance(obj, dict):

        for key, value in obj.items():

            child_path = (
                f"{path}.{key}"
            )

            _walk_checkpoint_631mc(
                value,
                path=child_path,
                depth=depth + 1,
                max_depth=max_depth,
            )

    elif isinstance(obj, list):

        # Plain scalar array.
        if len(obj) == FD003_ENDPOINT_N_631MC:

            if all(
                isinstance(
                    x,
                    (
                        bool,
                        int,
                        float,
                        np.integer,
                        np.floating,
                    ),
                )
                and not isinstance(
                    x,
                    complex,
                )
                for x in obj
            ):
                _record_array_candidate_631mc(
                    path,
                    obj,
                    "scalar_list",
                )

            # List of records: recover every scalar column.
            elif all(
                isinstance(x, dict)
                for x in obj
            ):

                common_keys = set(
                    obj[0].keys()
                )

                for rec in obj[1:]:
                    common_keys &= set(
                        rec.keys()
                    )

                for key in sorted(
                    common_keys
                ):

                    values = [
                        rec[key]
                        for rec in obj
                    ]

                    if all(
                        isinstance(
                            x,
                            (
                                bool,
                                int,
                                float,
                                np.integer,
                                np.floating,
                            ),
                        )
                        for x in values
                    ):
                        _record_array_candidate_631mc(
                            f"{path}[*].{key}",
                            values,
                            "record_column",
                        )

        # Continue recursion.
        for idx, value in enumerate(obj):

            if isinstance(
                value,
                (dict, list),
            ):
                _walk_checkpoint_631mc(
                    value,
                    path=f"{path}[{idx}]",
                    depth=depth + 1,
                    max_depth=max_depth,
                )


_walk_checkpoint_631mc(
    fd003_checkpoint_631mc
)


# Deduplicate by path.
_unique_fd003_arrays_631mc = {}

for row in fd003_checkpoint_arrays_631mc:
    _unique_fd003_arrays_631mc[
        row["path"]
    ] = row

fd003_checkpoint_arrays_631mc = sorted(
    _unique_fd003_arrays_631mc.values(),
    key=lambda x: x["path"],
)


# =============================================================================
# 9) FD003 — CRYPTOGRAPHIC HASH MATCH
# =============================================================================

fd003_checkpoint_exact_matches_631mc = []


for candidate in fd003_checkpoint_arrays_631mc:

    for semantic_key, expected_sha in (
        fd003_expected_hashes_631mc.items()
    ):

        for method, actual_sha in (
            candidate[
                "hash_variants"
            ].items()
        ):

            if actual_sha == expected_sha:

                fd003_checkpoint_exact_matches_631mc.append(
                    {
                        "semantic_key":
                            semantic_key,

                        "checkpoint_path":
                            candidate["path"],

                        "origin":
                            candidate["origin"],

                        "shape":
                            candidate["shape"],

                        "dtype":
                            candidate["dtype"],

                        "hash_method":
                            method,

                        "sha256":
                            actual_sha,
                    }
                )


fd003_checkpoint_exact_matches_631mc = sorted(
    fd003_checkpoint_exact_matches_631mc,
    key=lambda x: (
        x["semantic_key"],
        x["checkpoint_path"],
        x["hash_method"],
    ),
)


# =============================================================================
# 10) FD003 — STRUCTURAL CHECKPOINT INVENTORY
#
# Print keys and array paths only, not values.
# =============================================================================

def _structure_631mc(
    obj,
    prefix="root",
    depth=0,
    max_depth=5,
):
    rows = []

    if depth > max_depth:
        return rows

    if isinstance(obj, dict):

        rows.append(
            {
                "path": prefix,
                "type": "dict",
                "keys": [
                    str(k)
                    for k in obj.keys()
                ],
            }
        )

        for key, value in obj.items():

            if isinstance(
                value,
                (dict, list),
            ):
                rows.extend(
                    _structure_631mc(
                        value,
                        f"{prefix}.{key}",
                        depth + 1,
                        max_depth,
                    )
                )

    elif isinstance(obj, list):

        rows.append(
            {
                "path": prefix,
                "type": "list",
                "length": len(obj),
            }
        )

    return rows


fd003_checkpoint_structure_631mc = (
    _structure_631mc(
        fd003_checkpoint_631mc
    )
)


# =============================================================================
# 11) FREEZE RECOVERY AUDIT
# =============================================================================

FD001_FD003_PUBLICATION_RECOVERY_AUDIT_631MC = {
    "stage": "6.31M-C",

    "authority_audit_sha256":
        str(
            FD001_FD003_PUBLICATION_AUTHORITY_AUDIT_SHA256_631MB
        ),

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "risk_coverage_computed":
        False,

    "aurc_computed":
        False,

    "random_rejection_computed":
        False,

    "bootstrap_computed":
        False,

    "fd004_reopened":
        False,

    "fd001_final_root":
        str(fd001_final_root_631mc),

    "fd001_final_files":
        fd001_final_files_631mc,

    "fd001_final_evidence_file_sha256":
        _file_sha_631mc(
            fd001_final_evidence_path_631mc
        ),

    "fd001_final_evidence_metadata":
        fd001_final_evidence_metadata_631mc,

    "fd001_rowlevel_candidates":
        fd001_rowlevel_candidates_631mc,

    "fd003_checkpoint_path":
        str(fd003_checkpoint_path_631mc),

    "fd003_checkpoint_file_sha256":
        fd003_checkpoint_sha256_631mc,

    "fd003_endpoint_array_candidate_count":
        len(
            fd003_checkpoint_arrays_631mc
        ),

    "fd003_exact_hash_matches":
        fd003_checkpoint_exact_matches_631mc,
}


FD001_FD003_PUBLICATION_RECOVERY_AUDIT_SHA256_631MC = (
    _json_sha_631mc(
        FD001_FD003_PUBLICATION_RECOVERY_AUDIT_631MC
    )
)


all_checks_passed_631mc = True


# =============================================================================
# 12) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-C — FD001 FINAL-LINEAGE + "
    "FD003 CHECKPOINT PAYLOAD RECOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Risk–coverage / AURC computed     :", False)
print("Random rejection computed         :", False)
print("Bootstrap computed                :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


print(
    "\n--- FD001 FINAL DIRECTORY INVENTORY ---"
)

print(
    "Final file count:",
    len(fd001_final_files_631mc),
)

for row in fd001_final_files_631mc:

    print(
        "\n",
        row["path"],
    )

    print(
        "  size:",
        row.get("size_bytes"),
        "bytes",
    )

    print(
        "  SHA-256:",
        row.get("file_sha256"),
    )

    if "columns" in row:
        print(
            "  columns:",
            row["columns"],
        )

    if "keys" in row:
        print(
            "  NPZ keys:",
            row["keys"],
        )

    if "top_level_keys" in row:
        print(
            "  JSON top-level keys:",
            row["top_level_keys"],
        )


print(
    "\n--- FD001 FINAL EVIDENCE PROVENANCE FIELDS ---"
)

for row in fd001_final_evidence_metadata_631mc:

    if "value" in row:

        print(
            row["path"],
            "=",
            row["value"],
        )


print(
    "\n--- FD001 FINAL ROW-LEVEL CANDIDATES ---"
)

print(
    "Candidate count:",
    len(
        fd001_rowlevel_candidates_631mc
    ),
)

for row in fd001_rowlevel_candidates_631mc:

    print(
        "\n ",
        row["path"],
    )

    print(
        "   semantic hits:",
        row["semantic_hits"],
    )

    if "columns" in row:
        print(
            "   columns:",
            row["columns"],
        )

    if "keys" in row:
        print(
            "   keys:",
            row["keys"],
        )


print(
    "\n--- FD003 CHECKPOINT ---"
)

print(
    "Checkpoint path:",
    fd003_checkpoint_path_631mc,
)

print(
    "Checkpoint SHA-256:",
    fd003_checkpoint_sha256_631mc,
)

print(
    "\nTop checkpoint structure:"
)

for row in fd003_checkpoint_structure_631mc[:100]:

    if row["type"] == "dict":

        print(
            " ",
            row["path"],
            "| dict keys=",
            row["keys"],
        )

    else:

        print(
            " ",
            row["path"],
            "| list length=",
            row["length"],
        )


print(
    "\nFD003 endpoint-sized checkpoint arrays:",
    len(
        fd003_checkpoint_arrays_631mc
    ),
)

for row in fd003_checkpoint_arrays_631mc:

    print(
        " ",
        row["path"],
        "| origin=",
        row["origin"],
        "| shape=",
        row["shape"],
        "| dtype=",
        row["dtype"],
    )


print(
    "\n--- FD003 EXACT FROZEN HASH MATCHES ---"
)

print(
    "Exact match count:",
    len(
        fd003_checkpoint_exact_matches_631mc
    ),
)

for row in fd003_checkpoint_exact_matches_631mc:

    print(
        " ",
        row["semantic_key"],
        "->",
        row["checkpoint_path"],
        "| method=",
        row["hash_method"],
        "| shape=",
        row["shape"],
        "| dtype=",
        row["dtype"],
    )


print(
    "\nRecovery audit SHA-256:",
    FD001_FD003_PUBLICATION_RECOVERY_AUDIT_SHA256_631MC,
)

print(
    "All checks passed       :",
    all_checks_passed_631mc,
)

print(
    "\nPASS — final-lineage/checkpoint recovery completed."
)

print(
    "No publication-strengthening metric has been computed."
)

In [ ]:
# STEP 6.31M-D — FD003 LAST-MILE PERSISTENCE / NOTEBOOK-LINEAGE DISCOVERY
#
# PURPOSE
# -------
# Perform one final provenance-only search for FD003 held-out row-level evidence.
#
# We already know:
#   - FD003 contracts contain authoritative hashes;
#   - the restart checkpoint contains contracts/summaries only;
#   - no endpoint-sized arrays are currently resident.
#
# This step therefore asks:
#
#   1. Do historical notebook cells identify the exact FD003 endpoint-result
#      object names or persisted file paths?
#   2. Do any result files under 06_results contain a 100-row FD003-like
#      endpoint table/NPZ even if "fd003" is absent from the filename?
#   3. Do any safe text artifacts contain the exact frozen FD003 hashes and
#      point to a persisted row-level artifact?
#
# STRICTLY PROVENANCE / SCHEMA DISCOVERY.
#
# NO:
#   - raw TEST access
#   - raw RUL access
#   - model inference
#   - MC dropout
#   - prediction regeneration
#   - label reconstruction
#   - threshold/policy modification
#   - risk–coverage/AURC computation
#   - random rejection
#   - bootstrap
#   - new scientific metric

from pathlib import Path
import json
import hashlib
import re

import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631md = (
    "all_checks_passed_631l",
    "all_checks_passed_631ma",
    "all_checks_passed_631mb",
    "all_checks_passed_631mc",
    "FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L",
    "FD001_FD003_PUBLICATION_RECOVERY_AUDIT_SHA256_631MC",
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
)

missing_631md = [
    name
    for name in required_631md
    if name not in globals()
]

if missing_631md:
    raise RuntimeError(
        "STEP 6.31M-D missing required frozen state: "
        + ", ".join(missing_631md)
    )

if not bool(all_checks_passed_631l):
    raise RuntimeError("6.31L is not in PASS state.")

if not bool(all_checks_passed_631ma):
    raise RuntimeError("6.31M-A is not in PASS state.")

if not bool(all_checks_passed_631mb):
    raise RuntimeError("6.31M-B is not in PASS state.")

if not bool(all_checks_passed_631mc):
    raise RuntimeError("6.31M-C is not in PASS state.")

if not bool(FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L):
    raise RuntimeError("Empirical closure is not preserved.")

if not bool(FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L):
    raise RuntimeError("Notebook closure is not preserved.")


# =============================================================================
# 1) SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MD = False
RAW_TEST_ACCESS_631MD = False
RAW_RUL_ACCESS_631MD = False
MODEL_INFERENCE_631MD = False
MC_DROPOUT_631MD = False
PREDICTION_REGENERATION_631MD = False
LABEL_RECONSTRUCTION_631MD = False
THRESHOLD_TUNING_631MD = False
POLICY_SELECTION_631MD = False
RISK_COVERAGE_COMPUTED_631MD = False
AURC_COMPUTED_631MD = False
RANDOM_REJECTION_COMPUTED_631MD = False
BOOTSTRAP_COMPUTED_631MD = False
NEW_SCIENTIFIC_METRIC_631MD = False
FD004_REOPENED_631MD = False


# =============================================================================
# 2) HELPERS
# =============================================================================

def _json_sha_631md(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def _file_sha_631md(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def _safe_read_text_631md(path, max_bytes=25_000_000):
    """
    Read only text-like research/notebook/result artifacts.
    Raw dataset directory is never in scope.
    """
    path = Path(path)

    if path.stat().st_size > max_bytes:
        return None

    try:
        return path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        return None


# =============================================================================
# 3) AUTHORITATIVE FD003 HASHES / SEARCH TERMS
# =============================================================================

fd003_expected_hashes_631md = {}

for key in (
    "deterministic_prediction_sha256",
    "final_selective_risk_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
    "g1_risk_sha256",
    "phase_adjusted_uncertainty_sha256",
    "mc_mean_sha256",
    "mc_std_sha256",
    "endpoint_engine_ids_sha256",
    "endpoint_cycles_sha256",
):

    if key in FD003_TEST_FEATURE_SIDE_CONTRACT_629BA:
        fd003_expected_hashes_631md[key] = str(
            FD003_TEST_FEATURE_SIDE_CONTRACT_629BA[key]
        )


for key in (
    "absolute_error_sha256",
    "endpoint_results_sha256",
):

    if key in FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB:
        fd003_expected_hashes_631md[key] = str(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[key]
        )


fd003_search_terms_631md = [
    "FD003_TEST_FEATURE_SIDE_CONTRACT_629BA",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
    "629BA",
    "629BB",
    "endpoint_results_sha256",
    "absolute_error_sha256",
    "deterministic_prediction_sha256",
    "final_selective_risk_sha256",
    "accept_mask_sha256",
    "abstain_mask_sha256",
]

fd003_search_terms_631md.extend(
    fd003_expected_hashes_631md.values()
)

fd003_search_terms_631md = tuple(
    dict.fromkeys(fd003_search_terms_631md)
)


# =============================================================================
# 4) SAFE ROOTS ONLY
#
# Deliberately excludes:
#   C:\RUL_Reliability_Research\02_dataset
# =============================================================================

safe_roots_631md = (
    Path(r"C:\RUL_Reliability_Research\03_notebooks"),
    Path(r"C:\RUL_Reliability_Research\04_src"),
    Path(r"C:\RUL_Reliability_Research\05_experiments"),
    Path(r"C:\RUL_Reliability_Research\06_results"),
    Path(r"C:\RUL_Reliability_Research\08_paper"),
    Path(r"C:\RUL_Reliability_Research\09_notes"),
    Path(r"C:\RUL_Reliability_Research\checkpoints"),
)

text_suffixes_631md = {
    ".ipynb",
    ".json",
    ".txt",
    ".md",
    ".py",
    ".yaml",
    ".yml",
}


# =============================================================================
# 5) EXACT TEXT/HASH OCCURRENCE SEARCH
# =============================================================================

fd003_text_occurrences_631md = []

for root in safe_roots_631md:

    if not root.is_dir():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in text_suffixes_631md:
            continue

        text = _safe_read_text_631md(path)

        if text is None:
            continue

        matched_terms = [
            term
            for term in fd003_search_terms_631md
            if term in text
        ]

        if not matched_terms:
            continue

        fd003_text_occurrences_631md.append(
            {
                "path": str(path),
                "size_bytes": int(path.stat().st_size),
                "matched_terms": matched_terms,
            }
        )


fd003_text_occurrences_631md = sorted(
    fd003_text_occurrences_631md,
    key=lambda x: (
        -len(x["matched_terms"]),
        x["path"],
    ),
)


# =============================================================================
# 6) NOTEBOOK CODE-CELL LINEAGE SEARCH
#
# Read notebook JSON as text/code only.
# No notebook cells are executed.
# =============================================================================

fd003_notebook_cell_hits_631md = []

notebook_regex_631md = re.compile(
    r"(?i)"
    r"(FD003|629BA|629BB|"
    r"endpoint_results|absolute_error|"
    r"final_selective_risk|"
    r"deterministic_prediction|"
    r"accept_mask|abstain_mask|"
    r"to_csv|savez|np\.save|"
    r"06_results|checkpoint)"
)


for root in safe_roots_631md:

    if not root.is_dir():
        continue

    for path in root.rglob("*.ipynb"):

        try:
            with open(
                path,
                "r",
                encoding="utf-8",
            ) as f:
                nb = json.load(f)

        except Exception:
            continue

        for cell_index, cell in enumerate(
            nb.get("cells", [])
        ):

            if cell.get("cell_type") != "code":
                continue

            source = "".join(
                cell.get("source", [])
            )

            # Must be FD003/629BA/629BB relevant,
            # not merely a generic save call.
            source_low = source.lower()

            if not (
                "fd003" in source_low
                or "629ba" in source_low
                or "629bb" in source_low
            ):
                continue

            if not notebook_regex_631md.search(source):
                continue

            # Keep only relevant source lines.
            relevant_lines = []

            for line_no, line in enumerate(
                source.splitlines(),
                start=1,
            ):

                low = line.lower()

                if any(
                    token in low
                    for token in (
                        "fd003",
                        "629ba",
                        "629bb",
                        "endpoint_results",
                        "absolute_error",
                        "deterministic_prediction",
                        "final_selective_risk",
                        "accept_mask",
                        "abstain_mask",
                        "to_csv",
                        "savez",
                        "np.save",
                        "06_results",
                        "checkpoint",
                    )
                ):
                    relevant_lines.append(
                        {
                            "line": int(line_no),
                            "text": line.strip(),
                        }
                    )

            if relevant_lines:
                fd003_notebook_cell_hits_631md.append(
                    {
                        "notebook": str(path),
                        "cell_index": int(cell_index),
                        "relevant_lines":
                            relevant_lines[:120],
                    }
                )


fd003_notebook_cell_hits_631md = sorted(
    fd003_notebook_cell_hits_631md,
    key=lambda x: (
        x["notebook"],
        x["cell_index"],
    ),
)


# =============================================================================
# 7) SAFE 06_RESULTS SCHEMA SCAN FOR HIDDEN 100-ENDPOINT TABLES
#
# This intentionally does not rely on "fd003" being present in the filename.
#
# CSV:
#   - header must contain multiple endpoint-analysis semantic fields;
#   - if promising, count rows only.
#
# NPZ:
#   - inspect keys/shapes only;
#   - seek multiple 100-element endpoint arrays.
#
# No values are used to compute any metric.
# =============================================================================

results_root_631md = Path(
    r"C:\RUL_Reliability_Research\06_results"
)

semantic_tokens_631md = (
    "pred",
    "prediction",
    "risk",
    "reliab",
    "accept",
    "abstain",
    "decision",
    "error",
    "rul",
    "target",
    "truth",
    "engine",
    "unit",
    "endpoint",
)


fd003_hidden_csv_candidates_631md = []
fd003_hidden_npz_candidates_631md = []


if results_root_631md.is_dir():

    # -------------------------------------------------------------------------
    # CSV candidates
    # -------------------------------------------------------------------------
    for path in results_root_631md.rglob("*.csv"):

        try:
            header = pd.read_csv(
                path,
                nrows=0,
            )
        except Exception:
            continue

        cols = [
            str(c)
            for c in header.columns
        ]

        cols_low = [
            c.lower()
            for c in cols
        ]

        semantic_hits = sorted({
            token
            for token in semantic_tokens_631md
            if any(
                token in col
                for col in cols_low
            )
        })

        # Require substantial row-level endpoint semantics.
        if len(semantic_hits) < 3:
            continue

        try:
            # Metadata-only row count.
            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore",
            ) as f:
                row_count = max(
                    sum(1 for _ in f) - 1,
                    0,
                )
        except Exception:
            continue

        if row_count != 100:
            continue

        fd003_hidden_csv_candidates_631md.append(
            {
                "path": str(path),
                "size_bytes": int(path.stat().st_size),
                "row_count": int(row_count),
                "columns": cols,
                "semantic_hits": semantic_hits,
                "file_sha256": _file_sha_631md(path),
            }
        )

    # -------------------------------------------------------------------------
    # NPZ candidates
    # -------------------------------------------------------------------------
    for path in results_root_631md.rglob("*.npz"):

        try:
            with np.load(
                path,
                allow_pickle=False,
            ) as z:

                key_info = {}

                n_endpoint_arrays = 0

                semantic_hits = set()

                for key in z.files:

                    arr = z[key]

                    key_info[str(key)] = {
                        "shape": tuple(arr.shape),
                        "dtype": str(arr.dtype),
                    }

                    if (
                        arr.ndim >= 1
                        and arr.shape[0] == 100
                    ):
                        n_endpoint_arrays += 1

                    key_low = str(key).lower()

                    for token in semantic_tokens_631md:
                        if token in key_low:
                            semantic_hits.add(token)

                if (
                    n_endpoint_arrays < 2
                    or len(semantic_hits) < 2
                ):
                    continue

                fd003_hidden_npz_candidates_631md.append(
                    {
                        "path": str(path),
                        "size_bytes": int(path.stat().st_size),
                        "endpoint_array_count":
                            int(n_endpoint_arrays),
                        "keys": key_info,
                        "semantic_hits":
                            sorted(semantic_hits),
                        "file_sha256":
                            _file_sha_631md(path),
                    }
                )

        except Exception:
            continue


fd003_hidden_csv_candidates_631md = sorted(
    fd003_hidden_csv_candidates_631md,
    key=lambda x: (
        -len(x["semantic_hits"]),
        x["path"],
    ),
)

fd003_hidden_npz_candidates_631md = sorted(
    fd003_hidden_npz_candidates_631md,
    key=lambda x: (
        -x["endpoint_array_count"],
        -len(x["semantic_hits"]),
        x["path"],
    ),
)


# =============================================================================
# 8) EXTRACT SAFE RESULT-PATH STRINGS FROM FD003 CHECKPOINT/CONTRACTS
#
# We only retain strings referencing safe research-output locations.
# Dataset/raw paths are deliberately excluded.
# =============================================================================

fd003_safe_path_strings_631md = []


def _collect_safe_paths_631md(
    obj,
    origin,
    json_path="root",
):
    if isinstance(obj, dict):

        for key, value in obj.items():

            _collect_safe_paths_631md(
                value,
                origin,
                f"{json_path}.{key}",
            )

    elif isinstance(obj, list):

        for idx, value in enumerate(obj):

            _collect_safe_paths_631md(
                value,
                origin,
                f"{json_path}[{idx}]",
            )

    elif isinstance(obj, str):

        low = obj.lower()

        if (
            "rul_reliability_research" in low
            and any(
                safe_part in low
                for safe_part in (
                    "\\03_notebooks\\",
                    "\\04_src\\",
                    "\\05_experiments\\",
                    "\\06_results\\",
                    "\\08_paper\\",
                    "\\09_notes\\",
                    "\\checkpoints\\",
                )
            )
            and "\\02_dataset\\" not in low
        ):
            fd003_safe_path_strings_631md.append(
                {
                    "origin": origin,
                    "json_path": json_path,
                    "path_string": obj,
                }
            )


fd003_checkpoint_path_631md = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
    r"\fd003_post_629bc_restart_checkpoint_v1.json"
)

with open(
    fd003_checkpoint_path_631md,
    "r",
    encoding="utf-8",
) as f:
    fd003_checkpoint_obj_631md = json.load(f)


_collect_safe_paths_631md(
    fd003_checkpoint_obj_631md,
    "restart_checkpoint",
)

_collect_safe_paths_631md(
    FD003_TEST_FEATURE_SIDE_CONTRACT_629BA,
    "feature_side_contract_629BA",
)

_collect_safe_paths_631md(
    FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB,
    "heldout_score_contract_629BB",
)


# Deduplicate.
fd003_safe_path_strings_631md = sorted(
    {
        (
            x["origin"],
            x["json_path"],
            x["path_string"],
        )
        for x in fd003_safe_path_strings_631md
    }
)

fd003_safe_path_strings_631md = [
    {
        "origin": origin,
        "json_path": json_path,
        "path_string": path_string,
    }
    for origin, json_path, path_string
    in fd003_safe_path_strings_631md
]


# =============================================================================
# 9) FREEZE DISCOVERY AUDIT
# =============================================================================

FD003_LAST_MILE_PUBLICATION_DISCOVERY_631MD = {
    "stage": "6.31M-D",

    "recovery_audit_sha256":
        str(
            FD001_FD003_PUBLICATION_RECOVERY_AUDIT_SHA256_631MC
        ),

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "risk_coverage_computed":
        False,

    "aurc_computed":
        False,

    "random_rejection_computed":
        False,

    "bootstrap_computed":
        False,

    "fd004_reopened":
        False,

    "expected_fd003_hashes":
        fd003_expected_hashes_631md,

    "text_occurrence_count":
        len(fd003_text_occurrences_631md),

    "text_occurrences":
        fd003_text_occurrences_631md,

    "notebook_cell_hit_count":
        len(fd003_notebook_cell_hits_631md),

    "notebook_cell_hits":
        fd003_notebook_cell_hits_631md,

    "hidden_csv_candidate_count":
        len(fd003_hidden_csv_candidates_631md),

    "hidden_csv_candidates":
        fd003_hidden_csv_candidates_631md,

    "hidden_npz_candidate_count":
        len(fd003_hidden_npz_candidates_631md),

    "hidden_npz_candidates":
        fd003_hidden_npz_candidates_631md,

    "safe_path_string_count":
        len(fd003_safe_path_strings_631md),

    "safe_path_strings":
        fd003_safe_path_strings_631md,
}


FD003_LAST_MILE_PUBLICATION_DISCOVERY_SHA256_631MD = (
    _json_sha_631md(
        FD003_LAST_MILE_PUBLICATION_DISCOVERY_631MD
    )
)


all_checks_passed_631md = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-D — FD003 LAST-MILE PERSISTENCE / "
    "NOTEBOOK-LINEAGE DISCOVERY"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Risk–coverage / AURC computed     :", False)
print("Random rejection computed         :", False)
print("Bootstrap computed                :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


print(
    "\n--- EXACT HASH / CONTRACT TEXT OCCURRENCES ---"
)

print(
    "Artifact count:",
    len(fd003_text_occurrences_631md),
)

for row in fd003_text_occurrences_631md[:100]:

    print(
        "\n ",
        row["path"],
    )

    print(
        "   matched terms:",
        row["matched_terms"][:40],
    )


print(
    "\n--- FD003 NOTEBOOK CODE-CELL LINEAGE HITS ---"
)

print(
    "Cell hit count:",
    len(fd003_notebook_cell_hits_631md),
)

for hit in fd003_notebook_cell_hits_631md[:100]:

    print(
        "\nNotebook:",
        hit["notebook"],
    )

    print(
        "Cell index:",
        hit["cell_index"],
    )

    for line in hit["relevant_lines"]:

        print(
            f"  L{line['line']:>3}:",
            line["text"],
        )


print(
    "\n--- HIDDEN 100-ROW CSV CANDIDATES IN 06_results ---"
)

print(
    "Candidate count:",
    len(fd003_hidden_csv_candidates_631md),
)

for row in fd003_hidden_csv_candidates_631md[:100]:

    print(
        "\n ",
        row["path"],
    )

    print(
        "   semantic hits:",
        row["semantic_hits"],
    )

    print(
        "   columns:",
        row["columns"],
    )

    print(
        "   SHA-256:",
        row["file_sha256"],
    )


print(
    "\n--- HIDDEN 100-ENDPOINT NPZ CANDIDATES IN 06_results ---"
)

print(
    "Candidate count:",
    len(fd003_hidden_npz_candidates_631md),
)

for row in fd003_hidden_npz_candidates_631md[:100]:

    print(
        "\n ",
        row["path"],
    )

    print(
        "   endpoint arrays:",
        row["endpoint_array_count"],
    )

    print(
        "   semantic hits:",
        row["semantic_hits"],
    )

    print(
        "   keys:",
        row["keys"],
    )

    print(
        "   SHA-256:",
        row["file_sha256"],
    )


print(
    "\n--- SAFE FD003 PATH STRINGS IN FROZEN CONTRACTS/CHECKPOINT ---"
)

print(
    "Path-string count:",
    len(fd003_safe_path_strings_631md),
)

for row in fd003_safe_path_strings_631md[:100]:

    print(
        " ",
        row["origin"],
        "|",
        row["json_path"],
        "|",
        row["path_string"],
    )


print(
    "\nDiscovery audit SHA-256:",
    FD003_LAST_MILE_PUBLICATION_DISCOVERY_SHA256_631MD,
)

print(
    "All checks passed        :",
    all_checks_passed_631md,
)

print(
    "\nPASS — FD003 last-mile provenance discovery completed."
)

print(
    "No publication-strengthening metric has been computed."
)

In [ ]:
# STEP 6.31M-E — PUBLICATION-STRENGTHENING SELECTIVE ANALYSIS
#                FD001 + FD002 ONLY
#
# NEW REPORTING-ONLY ANALYSES
# ---------------------------
# 1. Full risk–coverage curve
# 2. Discrete MAE-AURC and normalized MAE-AURC
# 3. Matched random-rejection baseline at the ORIGINAL frozen coverage
# 4. Fixed-policy engine-level bootstrap confidence intervals
#
# IMPORTANT SCIENTIFIC BOUNDARY
# -----------------------------
# FD001 and FD002 use authoritative already-frozen row-level TEST evidence.
#
# FD003 is NOT reconstructed or rerun:
#   its original frozen negative held-out result remains authoritative.
#
# FD004 remains structurally unscored on held-out TEST.
#
# NONE of the analyses below may change:
#   - model
#   - risk architecture
#   - threshold
#   - accept/abstain decisions
#   - primary held-out results
#
# These are POST-HOC REPORTING ANALYSES ONLY.

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631me = (
    "all_checks_passed_631l",
    "all_checks_passed_631ma",
    "all_checks_passed_631mb",
    "all_checks_passed_631mc",
    "all_checks_passed_631md",
    "FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L",
    "FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L",
    "FD003_LAST_MILE_PUBLICATION_DISCOVERY_SHA256_631MD",
    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
)

missing_631me = [
    name
    for name in required_631me
    if name not in globals()
]

if missing_631me:
    raise RuntimeError(
        "STEP 6.31M-E missing required frozen state: "
        + ", ".join(missing_631me)
    )

if not bool(all_checks_passed_631l):
    raise RuntimeError("6.31L final closure is not PASS.")

if not bool(all_checks_passed_631md):
    raise RuntimeError("6.31M-D provenance discovery is not PASS.")

if not bool(FD001_FD004_PROJECT_EMPIRICAL_PHASE_CLOSED_631L):
    raise RuntimeError("Empirical closure is not preserved.")

if not bool(FD001_FD004_PROJECT_NOTEBOOK_WORKFLOW_CLOSED_631L):
    raise RuntimeError("Notebook closure is not preserved.")


# =============================================================================
# 1) SAFETY / ROLE DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631ME = False
RAW_TEST_ACCESS_631ME = False
RAW_RUL_ACCESS_631ME = False
MODEL_INFERENCE_631ME = False
MC_DROPOUT_631ME = False
PREDICTION_REGENERATION_631ME = False
LABEL_RECONSTRUCTION_631ME = False
THRESHOLD_TUNING_631ME = False
POLICY_SELECTION_631ME = False
PRIMARY_RESULT_CHANGED_631ME = False

POSTHOC_REPORTING_ANALYSIS_631ME = True

RISK_COVERAGE_COMPUTED_631ME = True
AURC_COMPUTED_631ME = True
MATCHED_RANDOM_REJECTION_COMPUTED_631ME = True
FIXED_POLICY_BOOTSTRAP_COMPUTED_631ME = True

FD003_RECONSTRUCTED_631ME = False
FD003_RERUN_631ME = False
FD004_REOPENED_631ME = False


# =============================================================================
# 2) HELPERS
# =============================================================================

def _file_sha256_631me(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def _q_631me(x, q):
    return float(
        np.quantile(
            np.asarray(x, dtype=np.float64),
            q,
        )
    )


def _risk_coverage_curve_631me(
    dataset,
    endpoint_id,
    risk,
    abs_error,
):
    """
    Discrete selective-risk curve.

    Lower reliability risk is better.

    Deterministic tie-breaking:
        (risk ascending, endpoint_id ascending)

    endpoint_id is outcome-independent, so ties are NEVER broken
    using error/label information.
    """

    endpoint_id = np.asarray(endpoint_id)
    risk = np.asarray(
        risk,
        dtype=np.float64,
    )
    abs_error = np.asarray(
        abs_error,
        dtype=np.float64,
    )

    n = int(len(risk))

    if not (
        len(endpoint_id) == n
        and len(abs_error) == n
    ):
        raise RuntimeError(
            f"{dataset}: risk-coverage vector length mismatch."
        )

    if n < 2:
        raise RuntimeError(
            f"{dataset}: insufficient endpoint population."
        )

    if not np.isfinite(risk).all():
        raise RuntimeError(
            f"{dataset}: non-finite reliability risk."
        )

    if not np.isfinite(abs_error).all():
        raise RuntimeError(
            f"{dataset}: non-finite absolute error."
        )

    # lexsort: last key is primary.
    order = np.lexsort(
        (
            endpoint_id,
            risk,
        )
    )

    sorted_id = endpoint_id[order]
    sorted_risk = risk[order]
    sorted_error = abs_error[order]

    cumulative_error = np.cumsum(
        sorted_error,
        dtype=np.float64,
    )

    k = np.arange(
        1,
        n + 1,
        dtype=np.int64,
    )

    coverage = (
        k.astype(np.float64)
        / float(n)
    )

    selective_mae = (
        cumulative_error
        / k.astype(np.float64)
    )

    full_mae = float(
        np.mean(abs_error)
    )

    curve = pd.DataFrame(
        {
            "dataset": dataset,
            "rank_k": k,
            "coverage": coverage,
            "endpoint_id_at_boundary":
                sorted_id,
            "risk_at_boundary":
                sorted_risk,
            "selective_mae":
                selective_mae,
            "full_mae":
                full_mae,
            "mae_gain":
                full_mae
                - selective_mae,
        }
    )

    # Standard discrete AURC analogue:
    # equal 1/N coverage increments -> mean selective loss
    # over retained sizes k=1,...,N.
    mae_aurc = float(
        selective_mae.mean()
    )

    normalized_mae_aurc = (
        float(mae_aurc / full_mae)
        if full_mae > 0.0
        else np.nan
    )

    # Under a random ranking, expected selective MAE at every k
    # equals full-population MAE. Therefore nAURC=1 is a useful
    # random-ranking reference.
    relative_aurc_improvement_vs_random = (
        float(
            1.0
            - normalized_mae_aurc
        )
        if np.isfinite(
            normalized_mae_aurc
        )
        else np.nan
    )

    unique_risk_count = int(
        np.unique(risk).size
    )

    tied_endpoint_count = int(
        n
        - unique_risk_count
    )

    summary = {
        "dataset": dataset,
        "n": n,
        "full_mae": full_mae,
        "mae_aurc": mae_aurc,
        "normalized_mae_aurc":
            normalized_mae_aurc,
        "relative_aurc_improvement_vs_random":
            relative_aurc_improvement_vs_random,
        "risk_unique_values":
            unique_risk_count,
        "risk_tied_endpoint_excess":
            tied_endpoint_count,
        "tie_break_rule":
            "risk ascending, then endpoint_id ascending",
    }

    return curve, summary


def _matched_random_rejection_631me(
    dataset,
    abs_error,
    frozen_accept,
    *,
    reps=20000,
    seed=631050,
    batch_size=500,
):
    """
    Compare the frozen accepted subset against random subsets of
    exactly the SAME size.

    This is a Monte-Carlo reporting baseline, not a retuned policy
    and not labeled as a formal hypothesis-test p-value.
    """

    abs_error = np.asarray(
        abs_error,
        dtype=np.float64,
    )

    frozen_accept = np.asarray(
        frozen_accept,
        dtype=bool,
    )

    n = int(len(abs_error))

    if len(frozen_accept) != n:
        raise RuntimeError(
            f"{dataset}: random-baseline mask length mismatch."
        )

    m = int(
        frozen_accept.sum()
    )

    if not (0 < m < n):
        raise RuntimeError(
            f"{dataset}: frozen accepted subset must be nonempty "
            "and smaller than the population."
        )

    observed_accepted_mae = float(
        np.mean(
            abs_error[
                frozen_accept
            ]
        )
    )

    full_mae = float(
        np.mean(abs_error)
    )

    rng = np.random.default_rng(
        int(seed)
    )

    random_mae = np.empty(
        int(reps),
        dtype=np.float64,
    )

    cursor = 0

    while cursor < int(reps):

        b = min(
            int(batch_size),
            int(reps) - cursor,
        )

        # Assign i.i.d. random keys and take the m smallest.
        # This samples uniformly among subsets of size m.
        random_keys = rng.random(
            (b, n)
        )

        idx = np.argpartition(
            random_keys,
            kth=m - 1,
            axis=1,
        )[:, :m]

        random_mae[
            cursor:cursor + b
        ] = (
            abs_error[idx].mean(
                axis=1
            )
        )

        cursor += b

    # Fraction of random matched-size subsets that do as well as
    # or better than the frozen reliability-selected subset.
    fraction_random_as_good_or_better = float(
        np.mean(
            random_mae
            <= observed_accepted_mae
        )
    )

    # Conservative Monte-Carlo smoothed version.
    mc_tail_smoothed = float(
        (
            np.count_nonzero(
                random_mae
                <= observed_accepted_mae
            )
            + 1
        )
        /
        (
            int(reps)
            + 1
        )
    )

    summary = {
        "dataset": dataset,
        "population_n": n,
        "frozen_accepted_n": m,
        "frozen_coverage":
            float(m / n),
        "full_mae":
            full_mae,
        "observed_frozen_accepted_mae":
            observed_accepted_mae,
        "random_replicates":
            int(reps),
        "random_seed":
            int(seed),
        "random_mae_mean":
            float(
                random_mae.mean()
            ),
        "random_mae_median":
            float(
                np.median(
                    random_mae
                )
            ),
        "random_mae_ci_2_5":
            _q_631me(
                random_mae,
                0.025,
            ),
        "random_mae_ci_97_5":
            _q_631me(
                random_mae,
                0.975,
            ),
        "random_median_minus_observed":
            float(
                np.median(
                    random_mae
                )
                - observed_accepted_mae
            ),
        "fraction_random_as_good_or_better":
            fraction_random_as_good_or_better,
        "mc_tail_smoothed":
            mc_tail_smoothed,
        "interpretation_boundary":
            (
                "Monte-Carlo matched-coverage reporting baseline; "
                "not a policy-selection criterion and not claimed "
                "as a model-training significance test."
            ),
    }

    return summary


def _fixed_policy_bootstrap_631me(
    dataset,
    abs_error,
    frozen_accept,
    *,
    reps=20000,
    seed=631051,
    batch_size=500,
):
    """
    Engine/endpoint bootstrap conditional on the already-frozen
    model, risk rule and accept mask.

    This quantifies held-out sample variability only.
    It does NOT quantify training-seed/model-fitting variability.
    """

    abs_error = np.asarray(
        abs_error,
        dtype=np.float64,
    )

    frozen_accept = np.asarray(
        frozen_accept,
        dtype=bool,
    )

    n = int(len(abs_error))

    if len(frozen_accept) != n:
        raise RuntimeError(
            f"{dataset}: bootstrap mask length mismatch."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    full_mae_rep = np.empty(
        int(reps),
        dtype=np.float64,
    )

    accepted_mae_rep = np.full(
        int(reps),
        np.nan,
        dtype=np.float64,
    )

    abstained_mae_rep = np.full(
        int(reps),
        np.nan,
        dtype=np.float64,
    )

    coverage_rep = np.empty(
        int(reps),
        dtype=np.float64,
    )

    gain_rep = np.full(
        int(reps),
        np.nan,
        dtype=np.float64,
    )

    separation_rep = np.full(
        int(reps),
        np.nan,
        dtype=np.float64,
    )

    cursor = 0

    while cursor < int(reps):

        b = min(
            int(batch_size),
            int(reps) - cursor,
        )

        sample_idx = rng.integers(
            0,
            n,
            size=(b, n),
        )

        err = abs_error[
            sample_idx
        ]

        acc = frozen_accept[
            sample_idx
        ]

        full = err.mean(
            axis=1
        )

        acc_n = acc.sum(
            axis=1
        )

        abs_n = (
            n
            - acc_n
        )

        cov = (
            acc_n.astype(
                np.float64
            )
            / float(n)
        )

        acc_sum = (
            err
            * acc
        ).sum(
            axis=1
        )

        abs_sum = (
            err
            * (~acc)
        ).sum(
            axis=1
        )

        valid_acc = (
            acc_n > 0
        )

        valid_abs = (
            abs_n > 0
        )

        acc_mae = np.full(
            b,
            np.nan,
            dtype=np.float64,
        )

        abs_mae = np.full(
            b,
            np.nan,
            dtype=np.float64,
        )

        acc_mae[
            valid_acc
        ] = (
            acc_sum[
                valid_acc
            ]
            /
            acc_n[
                valid_acc
            ]
        )

        abs_mae[
            valid_abs
        ] = (
            abs_sum[
                valid_abs
            ]
            /
            abs_n[
                valid_abs
            ]
        )

        gain = (
            full
            - acc_mae
        )

        separation = (
            abs_mae
            - acc_mae
        )

        sl = slice(
            cursor,
            cursor + b,
        )

        full_mae_rep[sl] = full
        accepted_mae_rep[sl] = acc_mae
        abstained_mae_rep[sl] = abs_mae
        coverage_rep[sl] = cov
        gain_rep[sl] = gain
        separation_rep[sl] = separation

        cursor += b

    finite_gain = np.isfinite(
        gain_rep
    )

    finite_sep = np.isfinite(
        separation_rep
    )

    if not finite_gain.any():
        raise RuntimeError(
            f"{dataset}: no valid bootstrap gain replicates."
        )

    if not finite_sep.any():
        raise RuntimeError(
            f"{dataset}: no valid bootstrap separation replicates."
        )

    observed_full_mae = float(
        abs_error.mean()
    )

    observed_accepted_mae = float(
        abs_error[
            frozen_accept
        ].mean()
    )

    observed_abstained_mae = float(
        abs_error[
            ~frozen_accept
        ].mean()
    )

    observed_gain = float(
        observed_full_mae
        - observed_accepted_mae
    )

    observed_separation = float(
        observed_abstained_mae
        - observed_accepted_mae
    )

    gain_valid = gain_rep[
        finite_gain
    ]

    sep_valid = separation_rep[
        finite_sep
    ]

    summary = {
        "dataset": dataset,
        "bootstrap_replicates":
            int(reps),
        "bootstrap_seed":
            int(seed),
        "population_n":
            n,
        "observed_coverage":
            float(
                frozen_accept.mean()
            ),
        "observed_full_mae":
            observed_full_mae,
        "observed_accepted_mae":
            observed_accepted_mae,
        "observed_abstained_mae":
            observed_abstained_mae,
        "observed_selective_mae_gain":
            observed_gain,
        "observed_abstained_minus_accepted_mae":
            observed_separation,

        "coverage_ci_2_5":
            _q_631me(
                coverage_rep,
                0.025,
            ),
        "coverage_ci_97_5":
            _q_631me(
                coverage_rep,
                0.975,
            ),

        "accepted_mae_ci_2_5":
            _q_631me(
                accepted_mae_rep[
                    np.isfinite(
                        accepted_mae_rep
                    )
                ],
                0.025,
            ),
        "accepted_mae_ci_97_5":
            _q_631me(
                accepted_mae_rep[
                    np.isfinite(
                        accepted_mae_rep
                    )
                ],
                0.975,
            ),

        "selective_gain_ci_2_5":
            _q_631me(
                gain_valid,
                0.025,
            ),
        "selective_gain_median":
            float(
                np.median(
                    gain_valid
                )
            ),
        "selective_gain_ci_97_5":
            _q_631me(
                gain_valid,
                0.975,
            ),
        "fraction_selective_gain_positive":
            float(
                np.mean(
                    gain_valid > 0.0
                )
            ),

        "separation_ci_2_5":
            _q_631me(
                sep_valid,
                0.025,
            ),
        "separation_median":
            float(
                np.median(
                    sep_valid
                )
            ),
        "separation_ci_97_5":
            _q_631me(
                sep_valid,
                0.975,
            ),
        "fraction_abstained_mae_higher":
            float(
                np.mean(
                    sep_valid > 0.0
                )
            ),

        "inference_boundary":
            (
                "Endpoint/bootstrap sampling variability conditional "
                "on the frozen trained model and frozen policy; "
                "does not represent model-training variability."
            ),
    }

    return summary


# =============================================================================
# 3) FD001 — LOAD AUTHORITATIVE FINAL ROW-LEVEL EVIDENCE
# =============================================================================

fd001_engine_path_631me = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\fd001_final\fd001_final_engine_results.csv"
)

fd001_evidence_path_631me = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\fd001_final\fd001_final_evidence.json"
)

if not fd001_engine_path_631me.is_file():
    raise RuntimeError(
        "FD001 final engine-level evidence is missing."
    )

if not fd001_evidence_path_631me.is_file():
    raise RuntimeError(
        "FD001 final evidence JSON is missing."
    )


# Verify FD001 row-level file has not changed since 6.31M-C.
fd001_mc_inventory_matches_631me = [
    row
    for row in fd001_final_files_631mc
    if Path(
        row["path"]
    ) == fd001_engine_path_631me
]

if len(
    fd001_mc_inventory_matches_631me
) != 1:
    raise RuntimeError(
        "Unable to uniquely recover FD001 engine-results "
        "fingerprint from 6.31M-C."
    )

expected_fd001_engine_sha_631me = (
    fd001_mc_inventory_matches_631me[0][
        "file_sha256"
    ]
)

actual_fd001_engine_sha_631me = (
    _file_sha256_631me(
        fd001_engine_path_631me
    )
)

if (
    actual_fd001_engine_sha_631me
    !=
    expected_fd001_engine_sha_631me
):
    raise RuntimeError(
        "FD001 final engine-results file changed after 6.31M-C."
    )


fd001_631me = pd.read_csv(
    fd001_engine_path_631me
)

with open(
    fd001_evidence_path_631me,
    "r",
    encoding="utf-8",
) as f:
    fd001_evidence_631me = json.load(f)


required_fd001_cols_631me = {
    "unit_id",
    "prediction",
    "selective_risk",
    "accept",
    "RUL_raw",
    "RUL_cap_125",
    "absolute_error",
}

missing_fd001_cols_631me = (
    required_fd001_cols_631me
    - set(fd001_631me.columns)
)

if missing_fd001_cols_631me:
    raise RuntimeError(
        "FD001 final table missing required columns: "
        + ", ".join(
            sorted(
                missing_fd001_cols_631me
            )
        )
    )

if len(fd001_631me) != 100:
    raise RuntimeError(
        "FD001 final engine table is not exactly 100 rows."
    )


fd001_prediction_631me = (
    fd001_631me[
        "prediction"
    ].to_numpy(
        dtype=np.float64
    )
)

fd001_true_631me = (
    fd001_631me[
        "RUL_cap_125"
    ].to_numpy(
        dtype=np.float64
    )
)

fd001_error_631me = (
    fd001_631me[
        "absolute_error"
    ].to_numpy(
        dtype=np.float64
    )
)

fd001_error_check_631me = np.abs(
    fd001_prediction_631me
    - fd001_true_631me
)

if not np.allclose(
    fd001_error_631me,
    fd001_error_check_631me,
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD001 stored absolute error does not exactly represent "
        "the final capped-RUL primary target."
    )


fd001_accept_631me = (
    fd001_631me[
        "accept"
    ].astype(
        bool
    ).to_numpy()
)

fd001_risk_631me = (
    fd001_631me[
        "selective_risk"
    ].to_numpy(
        dtype=np.float64
    )
)

fd001_id_631me = (
    fd001_631me[
        "unit_id"
    ].to_numpy()
)


# Frozen primary metric reproduction checks.
if int(
    fd001_accept_631me.sum()
) != int(
    fd001_evidence_631me[
        "accepted_engines"
    ]
):
    raise RuntimeError(
        "FD001 frozen accepted count changed."
    )

if not np.isclose(
    float(
        fd001_accept_631me.mean()
    ),
    float(
        fd001_evidence_631me[
            "coverage"
        ]
    ),
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD001 frozen coverage changed."
    )

if not np.isclose(
    float(
        fd001_error_631me.mean()
    ),
    float(
        fd001_evidence_631me[
            "primary_full_mae"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD001 full primary MAE no longer reproduces frozen evidence."
    )

if not np.isclose(
    float(
        fd001_error_631me[
            fd001_accept_631me
        ].mean()
    ),
    float(
        fd001_evidence_631me[
            "primary_accepted_mae"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD001 accepted primary MAE no longer reproduces frozen evidence."
    )


# =============================================================================
# 4) FD002 — LOAD AUTHORITATIVE C5 FROZEN SCORING EVIDENCE
# =============================================================================

fd002_root_631me = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_reliability_signals\fd002_c5"
)

fd002_score_path_631me = (
    fd002_root_631me
    / "fd002_c5_official_test_scoring.csv"
)

fd002_meta_path_631me = (
    fd002_root_631me
    / "fd002_c5_official_test_scoring_metadata.json"
)

fd002_decision_path_631me = (
    fd002_root_631me
    / "fd002_c5_test_operating_decisions.csv"
)

if not fd002_score_path_631me.is_file():
    raise RuntimeError(
        "FD002 official scoring CSV is missing."
    )

if not fd002_meta_path_631me.is_file():
    raise RuntimeError(
        "FD002 official scoring metadata is missing."
    )

if not fd002_decision_path_631me.is_file():
    raise RuntimeError(
        "FD002 frozen decision CSV is missing."
    )


fd002_631me = pd.read_csv(
    fd002_score_path_631me
)

fd002_decisions_631me = pd.read_csv(
    fd002_decision_path_631me
)

with open(
    fd002_meta_path_631me,
    "r",
    encoding="utf-8",
) as f:
    fd002_meta_631me = json.load(f)


if str(
    fd002_meta_631me.get(
        "status"
    )
) != "official_TEST_scored_and_frozen":
    raise RuntimeError(
        "FD002 scoring artifact is not the frozen official TEST authority."
    )

if bool(
    fd002_meta_631me.get(
        "threshold_tuned_after_test",
        True,
    )
):
    raise RuntimeError(
        "FD002 metadata indicates post-TEST threshold tuning."
    )

if bool(
    fd002_meta_631me.get(
        "threshold_modified_after_test",
        True,
    )
):
    raise RuntimeError(
        "FD002 metadata indicates post-TEST threshold modification."
    )

if bool(
    fd002_meta_631me.get(
        "model_retrained_after_test",
        True,
    )
):
    raise RuntimeError(
        "FD002 metadata indicates post-TEST model retraining."
    )


required_fd002_cols_631me = {
    "endpoint_position_1based",
    "evaluation_rul_capped_125",
    "point_prediction",
    "reliability_risk",
    "accepted",
    "abstained",
    "absolute_error",
}

missing_fd002_cols_631me = (
    required_fd002_cols_631me
    - set(fd002_631me.columns)
)

if missing_fd002_cols_631me:
    raise RuntimeError(
        "FD002 scoring table missing required columns: "
        + ", ".join(
            sorted(
                missing_fd002_cols_631me
            )
        )
    )

if len(fd002_631me) != 259:
    raise RuntimeError(
        "FD002 scoring table is not exactly 259 rows."
    )


fd002_prediction_631me = (
    fd002_631me[
        "point_prediction"
    ].to_numpy(
        dtype=np.float64
    )
)

fd002_true_631me = (
    fd002_631me[
        "evaluation_rul_capped_125"
    ].to_numpy(
        dtype=np.float64
    )
)

fd002_error_631me = (
    fd002_631me[
        "absolute_error"
    ].to_numpy(
        dtype=np.float64
    )
)

fd002_error_check_631me = np.abs(
    fd002_prediction_631me
    - fd002_true_631me
)

if not np.allclose(
    fd002_error_631me,
    fd002_error_check_631me,
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD002 absolute error does not reproduce capped TEST target."
    )


fd002_accept_631me = (
    fd002_631me[
        "accepted"
    ].astype(
        bool
    ).to_numpy()
)

fd002_abstain_631me = (
    fd002_631me[
        "abstained"
    ].astype(
        bool
    ).to_numpy()
)

if not np.array_equal(
    fd002_abstain_631me,
    ~fd002_accept_631me,
):
    raise RuntimeError(
        "FD002 accepted/abstained masks are not exact complements."
    )


fd002_risk_631me = (
    fd002_631me[
        "reliability_risk"
    ].to_numpy(
        dtype=np.float64
    )
)

fd002_id_631me = (
    fd002_631me[
        "endpoint_position_1based"
    ].to_numpy()
)


# Cross-check with pre-label operating decision table.
decision_key_631me = (
    "endpoint_position_1based"
)

merged_decision_check_631me = (
    fd002_631me[
        [
            decision_key_631me,
            "reliability_risk",
            "accepted",
            "abstained",
        ]
    ]
    .merge(
        fd002_decisions_631me[
            [
                decision_key_631me,
                "reliability_risk",
                "accepted",
                "abstained",
            ]
        ],
        on=decision_key_631me,
        how="inner",
        suffixes=(
            "_score",
            "_decision",
        ),
        validate="one_to_one",
    )
)

if len(
    merged_decision_check_631me
) != 259:
    raise RuntimeError(
        "FD002 decision/scoring endpoint alignment failed."
    )

if not np.allclose(
    merged_decision_check_631me[
        "reliability_risk_score"
    ].to_numpy(
        dtype=np.float64
    ),
    merged_decision_check_631me[
        "reliability_risk_decision"
    ].to_numpy(
        dtype=np.float64
    ),
    rtol=0.0,
    atol=0.0,
):
    raise RuntimeError(
        "FD002 pre-label risk changed between decision and scoring artifacts."
    )

if not np.array_equal(
    merged_decision_check_631me[
        "accepted_score"
    ].astype(bool).to_numpy(),
    merged_decision_check_631me[
        "accepted_decision"
    ].astype(bool).to_numpy(),
):
    raise RuntimeError(
        "FD002 pre-label accepted mask changed during scoring."
    )


if int(
    fd002_accept_631me.sum()
) != int(
    fd002_meta_631me[
        "accepted_test_endpoints"
    ]
):
    raise RuntimeError(
        "FD002 frozen accepted count changed."
    )

if not np.isclose(
    float(
        fd002_accept_631me.mean()
    ),
    float(
        fd002_meta_631me[
            "realized_test_coverage"
        ]
    ),
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError(
        "FD002 frozen coverage changed."
    )

if not np.isclose(
    float(
        fd002_error_631me.mean()
    ),
    float(
        fd002_meta_631me[
            "full_test_mae"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD002 full MAE no longer reproduces frozen authority."
    )

if not np.isclose(
    float(
        fd002_error_631me[
            fd002_accept_631me
        ].mean()
    ),
    float(
        fd002_meta_631me[
            "accepted_test_mae"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
):
    raise RuntimeError(
        "FD002 accepted MAE no longer reproduces frozen authority."
    )


# =============================================================================
# 5) RISK–COVERAGE / AURC
# =============================================================================

(
    FD001_PUBLICATION_RISK_COVERAGE_CURVE_631ME,
    fd001_aurc_summary_631me,
) = _risk_coverage_curve_631me(
    "FD001_capped_RUL",
    fd001_id_631me,
    fd001_risk_631me,
    fd001_error_631me,
)


(
    FD002_PUBLICATION_RISK_COVERAGE_CURVE_631ME,
    fd002_aurc_summary_631me,
) = _risk_coverage_curve_631me(
    "FD002_capped_RUL",
    fd002_id_631me,
    fd002_risk_631me,
    fd002_error_631me,
)


FD001_FD002_PUBLICATION_AURC_SUMMARY_631ME = (
    pd.DataFrame(
        [
            fd001_aurc_summary_631me,
            fd002_aurc_summary_631me,
        ]
    )
)


# =============================================================================
# 6) MATCHED RANDOM-REJECTION BASELINE
# =============================================================================

fd001_random_summary_631me = (
    _matched_random_rejection_631me(
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        reps=20000,
        seed=631051,
    )
)


fd002_random_summary_631me = (
    _matched_random_rejection_631me(
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        reps=20000,
        seed=631052,
    )
)


FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631ME = (
    pd.DataFrame(
        [
            fd001_random_summary_631me,
            fd002_random_summary_631me,
        ]
    )
)


# =============================================================================
# 7) FIXED-POLICY BOOTSTRAP
# =============================================================================

fd001_bootstrap_summary_631me = (
    _fixed_policy_bootstrap_631me(
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        reps=20000,
        seed=631061,
    )
)


fd002_bootstrap_summary_631me = (
    _fixed_policy_bootstrap_631me(
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        reps=20000,
        seed=631062,
    )
)


FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631ME = (
    pd.DataFrame(
        [
            fd001_bootstrap_summary_631me,
            fd002_bootstrap_summary_631me,
        ]
    )
)


# =============================================================================
# 8) PRESERVE FD003 ORIGINAL NEGATIVE RESULT — NO NEW CALCULATION
# =============================================================================

FD003_PUBLICATION_STRENGTHENING_STATUS_631ME = {
    "dataset": "FD003",
    "row_level_posthoc_analysis_performed":
        False,
    "reason":
        (
            "authoritative held-out row-level arrays were not "
            "persisted/recoverable without reconstruction or rerun"
        ),
    "original_frozen_result_preserved":
        True,
    "endpoint_count":
        int(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "endpoint_count"
            ]
        ),
    "realized_coverage":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "realized_coverage"
            ]
        ),
    "full_mae":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "full_mae"
            ]
        ),
    "accepted_mae":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "accepted_mae"
            ]
        ),
    "selective_mae_gain":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "selective_mae_gain"
            ]
        ),
    "interpretation":
        (
            "original frozen held-out negative result retained; "
            "no post-hoc row-level reconstruction performed"
        ),
}


# =============================================================================
# 9) COMPACT PUBLICATION SUMMARY
# =============================================================================

publication_rows_631me = []

for (
    dataset,
    error,
    accept,
    aurc,
    random_summary,
    bootstrap_summary,
) in (
    (
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        fd001_aurc_summary_631me,
        fd001_random_summary_631me,
        fd001_bootstrap_summary_631me,
    ),
    (
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        fd002_aurc_summary_631me,
        fd002_random_summary_631me,
        fd002_bootstrap_summary_631me,
    ),
):

    publication_rows_631me.append(
        {
            "dataset":
                dataset,

            "n":
                int(len(error)),

            "frozen_coverage":
                float(
                    np.mean(
                        accept
                    )
                ),

            "full_mae":
                float(
                    np.mean(
                        error
                    )
                ),

            "frozen_accepted_mae":
                float(
                    np.mean(
                        error[
                            accept
                        ]
                    )
                ),

            "frozen_mae_gain":
                float(
                    np.mean(
                        error
                    )
                    -
                    np.mean(
                        error[
                            accept
                        ]
                    )
                ),

            "normalized_mae_aurc":
                float(
                    aurc[
                        "normalized_mae_aurc"
                    ]
                ),

            "aurc_relative_improvement_vs_random":
                float(
                    aurc[
                        "relative_aurc_improvement_vs_random"
                    ]
                ),

            "matched_random_median_mae":
                float(
                    random_summary[
                        "random_mae_median"
                    ]
                ),

            "matched_random_median_minus_observed":
                float(
                    random_summary[
                        "random_median_minus_observed"
                    ]
                ),

            "random_fraction_as_good_or_better":
                float(
                    random_summary[
                        "fraction_random_as_good_or_better"
                    ]
                ),

            "bootstrap_gain_ci_low":
                float(
                    bootstrap_summary[
                        "selective_gain_ci_2_5"
                    ]
                ),

            "bootstrap_gain_ci_high":
                float(
                    bootstrap_summary[
                        "selective_gain_ci_97_5"
                    ]
                ),

            "bootstrap_fraction_gain_positive":
                float(
                    bootstrap_summary[
                        "fraction_selective_gain_positive"
                    ]
                ),

            "bootstrap_separation_ci_low":
                float(
                    bootstrap_summary[
                        "separation_ci_2_5"
                    ]
                ),

            "bootstrap_separation_ci_high":
                float(
                    bootstrap_summary[
                        "separation_ci_97_5"
                    ]
                ),

            "bootstrap_fraction_abstained_mae_higher":
                float(
                    bootstrap_summary[
                        "fraction_abstained_mae_higher"
                    ]
                ),
        }
    )


FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631ME = (
    pd.DataFrame(
        publication_rows_631me
    )
)


# =============================================================================
# 10) AUDIT CONTRACT — IN MEMORY ONLY FOR NOW
# =============================================================================

FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631ME = {
    "stage":
        "6.31M-E",

    "role":
        "posthoc_reporting_only_publication_strengthening",

    "upstream_discovery_sha256":
        str(
            FD003_LAST_MILE_PUBLICATION_DISCOVERY_SHA256_631MD
        ),

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "primary_result_changed":
        False,

    "fd001_primary_target":
        "RUL_cap_125",

    "fd002_evaluation_target":
        "piecewise-capped RUL, cap=125",

    "fd003_new_row_level_analysis":
        False,

    "fd003_original_frozen_negative_result_preserved":
        True,

    "fd004_reopened":
        False,

    "risk_coverage_definition":
        (
            "ascending frozen reliability risk; deterministic "
            "outcome-independent endpoint-ID tie break"
        ),

    "aurc_definition":
        (
            "mean retained-subset MAE over k=1..N "
            "(equal 1/N coverage increments)"
        ),

    "matched_random_baseline":
        (
            "random subsets with exactly the frozen accepted count"
        ),

    "bootstrap_scope":
        (
            "endpoint sampling variability conditional on frozen "
            "trained model and frozen operating policy"
        ),

    "formal_training_significance_claim":
        False,
}


def _json_sha_631me(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631ME = (
    _json_sha_631me(
        FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631ME
    )
)


all_checks_passed_631me = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-E — PUBLICATION-STRENGTHENING "
    "SELECTIVE ANALYSIS (FD001 + FD002)"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Primary held-out results changed  :", False)
print("Post-hoc reporting analysis       :", True)
print("Risk–coverage / AURC              :", True)
print("Matched random rejection          :", True)
print("Fixed-policy bootstrap            :", True)
print("FD003 reconstructed/rerun         :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


print(
    "\n--- PUBLICATION-STRENGTHENING SUMMARY ---"
)

print(
    FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631ME.to_string(
        index=False
    )
)


print(
    "\n--- AURC DETAIL ---"
)

print(
    FD001_FD002_PUBLICATION_AURC_SUMMARY_631ME.to_string(
        index=False
    )
)


print(
    "\n--- MATCHED RANDOM-REJECTION BASELINE ---"
)

print(
    FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631ME.to_string(
        index=False
    )
)


print(
    "\n--- FIXED-POLICY BOOTSTRAP ---"
)

print(
    FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631ME.to_string(
        index=False
    )
)


print(
    "\n--- FD003 PRESERVATION STATUS ---"
)

print(
    json.dumps(
        FD003_PUBLICATION_STRENGTHENING_STATUS_631ME,
        indent=2,
    )
)


print(
    "\nPublication-analysis contract SHA-256:",
    FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631ME,
)

print(
    "All checks passed                     :",
    all_checks_passed_631me,
)

print(
    "\nPASS — reporting-only publication-strengthening analyses "
    "completed without changing any frozen primary result."
)

print(
    "Nothing has been persisted/frozen yet; inspect these results "
    "before creating publication artifacts."
)

In [ ]:
# STEP 6.31M-E-R1 — FD002 DECISION/SCORING RISK PRECISION + IDENTITY AUDIT
#
# PURPOSE
# -------
# Diagnose why the frozen pre-label decision CSV and the official scoring CSV
# are not bit-for-bit equal in their reliability_risk column.
#
# THIS STEP COMPUTES NO PUBLICATION METRIC.
#
# It checks:
#   1. endpoint ordering/alignment,
#   2. exact and numerical risk differences,
#   3. frozen accept/abstain mask identity,
#   4. threshold-rule reproduction in BOTH artifacts,
#   5. whether any risk difference can change a decision,
#   6. whether risk ordering/ranking is preserved.
#
# NO:
#   - model inference
#   - MC dropout
#   - TEST/RUL raw access
#   - retuning
#   - new threshold
#   - AURC
#   - bootstrap
#   - random rejection

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


# =============================================================================
# 0) PATHS
# =============================================================================

fd002_root_631mer1 = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\step6_reliability_signals\fd002_c5"
)

score_path_631mer1 = (
    fd002_root_631mer1
    / "fd002_c5_official_test_scoring.csv"
)

decision_path_631mer1 = (
    fd002_root_631mer1
    / "fd002_c5_test_operating_decisions.csv"
)

score_meta_path_631mer1 = (
    fd002_root_631mer1
    / "fd002_c5_official_test_scoring_metadata.json"
)

decision_meta_path_631mer1 = (
    fd002_root_631mer1
    / "fd002_c5_test_operating_decisions_metadata.json"
)

for p_631mer1 in (
    score_path_631mer1,
    decision_path_631mer1,
    score_meta_path_631mer1,
    decision_meta_path_631mer1,
):
    if not p_631mer1.is_file():
        raise RuntimeError(
            "Missing required frozen FD002 artifact: "
            + str(p_631mer1)
        )


# =============================================================================
# 1) LOAD FROZEN ARTIFACTS ONLY
# =============================================================================

score_631mer1 = pd.read_csv(
    score_path_631mer1
)

decision_631mer1 = pd.read_csv(
    decision_path_631mer1
)

with open(
    score_meta_path_631mer1,
    "r",
    encoding="utf-8",
) as f:
    score_meta_631mer1 = json.load(f)

with open(
    decision_meta_path_631mer1,
    "r",
    encoding="utf-8",
) as f:
    decision_meta_631mer1 = json.load(f)


required_cols_631mer1 = {
    "endpoint_position_1based",
    "reliability_risk",
    "accepted",
    "abstained",
}

for label_631mer1, df_631mer1 in (
    ("scoring", score_631mer1),
    ("decision", decision_631mer1),
):
    missing_631mer1 = (
        required_cols_631mer1
        - set(df_631mer1.columns)
    )

    if missing_631mer1:
        raise RuntimeError(
            f"FD002 {label_631mer1} artifact missing columns: "
            + ", ".join(
                sorted(missing_631mer1)
            )
        )


# =============================================================================
# 2) ALIGN BY FROZEN ENDPOINT ID
# =============================================================================

merged_631mer1 = (
    score_631mer1[
        [
            "endpoint_position_1based",
            "reliability_risk",
            "accepted",
            "abstained",
        ]
    ]
    .merge(
        decision_631mer1[
            [
                "endpoint_position_1based",
                "reliability_risk",
                "accepted",
                "abstained",
            ]
        ],
        on="endpoint_position_1based",
        how="inner",
        suffixes=(
            "_score",
            "_decision",
        ),
        validate="one_to_one",
    )
    .sort_values(
        "endpoint_position_1based"
    )
    .reset_index(drop=True)
)

if len(merged_631mer1) != 259:
    raise RuntimeError(
        "FD002 scoring/decision endpoint alignment is not exactly 259."
    )


# =============================================================================
# 3) RISK DIFFERENCE AUDIT
# =============================================================================

risk_score_631mer1 = (
    merged_631mer1[
        "reliability_risk_score"
    ].to_numpy(dtype=np.float64)
)

risk_decision_631mer1 = (
    merged_631mer1[
        "reliability_risk_decision"
    ].to_numpy(dtype=np.float64)
)

risk_delta_631mer1 = (
    risk_score_631mer1
    - risk_decision_631mer1
)

risk_abs_delta_631mer1 = np.abs(
    risk_delta_631mer1
)

exact_risk_equal_631mer1 = bool(
    np.array_equal(
        risk_score_631mer1,
        risk_decision_631mer1,
    )
)

max_abs_risk_diff_631mer1 = float(
    risk_abs_delta_631mer1.max()
)

mean_abs_risk_diff_631mer1 = float(
    risk_abs_delta_631mer1.mean()
)

nonzero_risk_diff_count_631mer1 = int(
    np.count_nonzero(
        risk_abs_delta_631mer1 > 0.0
    )
)

risk_diff_quantiles_631mer1 = {
    "q50": float(
        np.quantile(
            risk_abs_delta_631mer1,
            0.50,
        )
    ),
    "q90": float(
        np.quantile(
            risk_abs_delta_631mer1,
            0.90,
        )
    ),
    "q99": float(
        np.quantile(
            risk_abs_delta_631mer1,
            0.99,
        )
    ),
    "q100": max_abs_risk_diff_631mer1,
}


# =============================================================================
# 4) MASK IDENTITY
# =============================================================================

accept_score_631mer1 = (
    merged_631mer1[
        "accepted_score"
    ].astype(bool).to_numpy()
)

accept_decision_631mer1 = (
    merged_631mer1[
        "accepted_decision"
    ].astype(bool).to_numpy()
)

abstain_score_631mer1 = (
    merged_631mer1[
        "abstained_score"
    ].astype(bool).to_numpy()
)

abstain_decision_631mer1 = (
    merged_631mer1[
        "abstained_decision"
    ].astype(bool).to_numpy()
)

accept_exact_equal_631mer1 = bool(
    np.array_equal(
        accept_score_631mer1,
        accept_decision_631mer1,
    )
)

abstain_exact_equal_631mer1 = bool(
    np.array_equal(
        abstain_score_631mer1,
        abstain_decision_631mer1,
    )
)

accept_mismatch_count_631mer1 = int(
    np.count_nonzero(
        accept_score_631mer1
        != accept_decision_631mer1
    )
)

abstain_mismatch_count_631mer1 = int(
    np.count_nonzero(
        abstain_score_631mer1
        != abstain_decision_631mer1
    )
)

if not np.array_equal(
    abstain_score_631mer1,
    ~accept_score_631mer1,
):
    raise RuntimeError(
        "Scoring artifact accept/abstain masks are not complements."
    )

if not np.array_equal(
    abstain_decision_631mer1,
    ~accept_decision_631mer1,
):
    raise RuntimeError(
        "Decision artifact accept/abstain masks are not complements."
    )


# =============================================================================
# 5) RECOVER FROZEN THRESHOLD
# =============================================================================

threshold_candidates_631mer1 = []

for source_name_631mer1, meta_631mer1 in (
    ("scoring_metadata", score_meta_631mer1),
    ("decision_metadata", decision_meta_631mer1),
):
    for key_631mer1 in (
        "frozen_exact_risk_threshold",
        "frozen_risk_threshold",
        "risk_threshold",
        "threshold",
    ):
        if key_631mer1 in meta_631mer1:
            value_631mer1 = meta_631mer1[
                key_631mer1
            ]

            if isinstance(
                value_631mer1,
                (int, float),
            ):
                threshold_candidates_631mer1.append(
                    (
                        source_name_631mer1,
                        key_631mer1,
                        float(value_631mer1),
                    )
                )

if not threshold_candidates_631mer1:
    raise RuntimeError(
        "Unable to recover frozen FD002 risk threshold from metadata."
    )

unique_thresholds_631mer1 = sorted({
    round(x[2], 15)
    for x in threshold_candidates_631mer1
})

if len(unique_thresholds_631mer1) != 1:
    raise RuntimeError(
        "FD002 scoring and decision metadata disagree on frozen threshold: "
        + repr(threshold_candidates_631mer1)
    )

frozen_threshold_631mer1 = float(
    threshold_candidates_631mer1[0][2]
)


# =============================================================================
# 6) REPRODUCE FROZEN DECISIONS FROM EACH RISK REPRESENTATION
# =============================================================================

accept_from_score_risk_631mer1 = (
    risk_score_631mer1
    <= frozen_threshold_631mer1
)

accept_from_decision_risk_631mer1 = (
    risk_decision_631mer1
    <= frozen_threshold_631mer1
)

score_rule_exact_631mer1 = bool(
    np.array_equal(
        accept_from_score_risk_631mer1,
        accept_score_631mer1,
    )
)

decision_rule_exact_631mer1 = bool(
    np.array_equal(
        accept_from_decision_risk_631mer1,
        accept_decision_631mer1,
    )
)

cross_rule_exact_631mer1 = bool(
    np.array_equal(
        accept_from_score_risk_631mer1,
        accept_from_decision_risk_631mer1,
    )
)

score_rule_mismatch_count_631mer1 = int(
    np.count_nonzero(
        accept_from_score_risk_631mer1
        != accept_score_631mer1
    )
)

decision_rule_mismatch_count_631mer1 = int(
    np.count_nonzero(
        accept_from_decision_risk_631mer1
        != accept_decision_631mer1
    )
)


# =============================================================================
# 7) THRESHOLD-MARGIN SAFETY CHECK
# =============================================================================

score_margin_631mer1 = np.abs(
    risk_score_631mer1
    - frozen_threshold_631mer1
)

decision_margin_631mer1 = np.abs(
    risk_decision_631mer1
    - frozen_threshold_631mer1
)

minimum_threshold_margin_631mer1 = float(
    min(
        score_margin_631mer1.min(),
        decision_margin_631mer1.min(),
    )
)

difference_vs_margin_ratio_631mer1 = (
    float(
        max_abs_risk_diff_631mer1
        / minimum_threshold_margin_631mer1
    )
    if minimum_threshold_margin_631mer1 > 0
    else np.inf
)


# =============================================================================
# 8) RANK / ORDER IDENTITY
#
# Important for later risk–coverage analysis.
# =============================================================================

ids_631mer1 = (
    merged_631mer1[
        "endpoint_position_1based"
    ].to_numpy()
)

order_score_631mer1 = np.lexsort(
    (
        ids_631mer1,
        risk_score_631mer1,
    )
)

order_decision_631mer1 = np.lexsort(
    (
        ids_631mer1,
        risk_decision_631mer1,
    )
)

risk_order_exact_631mer1 = bool(
    np.array_equal(
        order_score_631mer1,
        order_decision_631mer1,
    )
)

risk_order_mismatch_positions_631mer1 = int(
    np.count_nonzero(
        order_score_631mer1
        != order_decision_631mer1
    )
)


# =============================================================================
# 9) NUMERICAL-TOLERANCE CLASSIFICATION
# =============================================================================

machine_eps_631mer1 = float(
    np.finfo(np.float64).eps
)

# Diagnostic only — no scientific decision is based on this tolerance.
roundtrip_level_631mer1 = bool(
    max_abs_risk_diff_631mer1
    <= 1e-12
)

decision_invariant_631mer1 = bool(
    accept_exact_equal_631mer1
    and abstain_exact_equal_631mer1
    and score_rule_exact_631mer1
    and decision_rule_exact_631mer1
    and cross_rule_exact_631mer1
)

ranking_invariant_631mer1 = bool(
    risk_order_exact_631mer1
)

safe_serialization_difference_631mer1 = bool(
    roundtrip_level_631mer1
    and decision_invariant_631mer1
    and ranking_invariant_631mer1
)


# =============================================================================
# 10) AUDIT OBJECT
# =============================================================================

FD002_RISK_SERIALIZATION_AUDIT_631MER1 = {
    "stage":
        "6.31M-E-R1",

    "scientific_recomputation":
        False,

    "publication_metric_computed":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "threshold_changed":
        False,

    "policy_changed":
        False,

    "endpoint_count":
        int(len(merged_631mer1)),

    "frozen_threshold":
        frozen_threshold_631mer1,

    "threshold_candidates":
        threshold_candidates_631mer1,

    "risk_exact_equal":
        exact_risk_equal_631mer1,

    "risk_nonzero_difference_count":
        nonzero_risk_diff_count_631mer1,

    "risk_max_abs_difference":
        max_abs_risk_diff_631mer1,

    "risk_mean_abs_difference":
        mean_abs_risk_diff_631mer1,

    "risk_abs_difference_quantiles":
        risk_diff_quantiles_631mer1,

    "float64_machine_epsilon":
        machine_eps_631mer1,

    "accept_masks_exact_equal":
        accept_exact_equal_631mer1,

    "abstain_masks_exact_equal":
        abstain_exact_equal_631mer1,

    "accept_mask_mismatch_count":
        accept_mismatch_count_631mer1,

    "abstain_mask_mismatch_count":
        abstain_mismatch_count_631mer1,

    "score_csv_reproduces_frozen_rule":
        score_rule_exact_631mer1,

    "decision_csv_reproduces_frozen_rule":
        decision_rule_exact_631mer1,

    "risk_representations_produce_same_decisions":
        cross_rule_exact_631mer1,

    "score_rule_mismatch_count":
        score_rule_mismatch_count_631mer1,

    "decision_rule_mismatch_count":
        decision_rule_mismatch_count_631mer1,

    "minimum_distance_to_frozen_threshold":
        minimum_threshold_margin_631mer1,

    "max_difference_to_threshold_margin_ratio":
        difference_vs_margin_ratio_631mer1,

    "risk_order_exact":
        risk_order_exact_631mer1,

    "risk_order_mismatch_positions":
        risk_order_mismatch_positions_631mer1,

    "roundtrip_level_le_1e_12":
        roundtrip_level_631mer1,

    "decision_invariant":
        decision_invariant_631mer1,

    "ranking_invariant":
        ranking_invariant_631mer1,

    "safe_serialization_difference":
        safe_serialization_difference_631mer1,
}


def _sha_json_631mer1(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1 = (
    _sha_json_631mer1(
        FD002_RISK_SERIALIZATION_AUDIT_631MER1
    )
)


all_checks_passed_631mer1 = True


# =============================================================================
# 11) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-E-R1 — FD002 DECISION/SCORING "
    "RISK PRECISION + IDENTITY AUDIT"
)

print("=" * 176)

print("Publication metric computed       :", False)
print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Model inference / MC dropout      :", False)
print("Threshold changed                 :", False)
print("Policy changed                    :", False)

print("\n--- FROZEN THRESHOLD ---")
print(
    "Threshold:",
    repr(frozen_threshold_631mer1)
)
print(
    "Threshold metadata candidates:",
    threshold_candidates_631mer1
)

print("\n--- RISK NUMERICAL IDENTITY ---")
print(
    "Exact risk equality              :",
    exact_risk_equal_631mer1
)
print(
    "Nonzero risk differences         :",
    nonzero_risk_diff_count_631mer1,
    "/",
    len(merged_631mer1),
)
print(
    "Maximum absolute risk difference :",
    f"{max_abs_risk_diff_631mer1:.18e}"
)
print(
    "Mean absolute risk difference    :",
    f"{mean_abs_risk_diff_631mer1:.18e}"
)
print(
    "Difference quantiles             :",
    risk_diff_quantiles_631mer1
)
print(
    "float64 machine epsilon          :",
    f"{machine_eps_631mer1:.18e}"
)

print("\n--- MASK IDENTITY ---")
print(
    "Accept masks exactly equal       :",
    accept_exact_equal_631mer1
)
print(
    "Accept mismatches                :",
    accept_mismatch_count_631mer1
)
print(
    "Abstain masks exactly equal      :",
    abstain_exact_equal_631mer1
)
print(
    "Abstain mismatches               :",
    abstain_mismatch_count_631mer1
)

print("\n--- FROZEN RULE REPRODUCTION ---")
print(
    "Scoring risk -> stored mask      :",
    score_rule_exact_631mer1
)
print(
    "Decision risk -> stored mask     :",
    decision_rule_exact_631mer1
)
print(
    "Both risk versions same decision :",
    cross_rule_exact_631mer1
)

print("\n--- THRESHOLD MARGIN ---")
print(
    "Minimum risk distance to threshold:",
    f"{minimum_threshold_margin_631mer1:.18e}"
)
print(
    "Max risk diff / min margin ratio :",
    f"{difference_vs_margin_ratio_631mer1:.18e}"
)

print("\n--- RISK ORDERING ---")
print(
    "Risk ordering exactly identical  :",
    risk_order_exact_631mer1
)
print(
    "Ordering mismatch positions      :",
    risk_order_mismatch_positions_631mer1
)

print("\n--- DIAGNOSTIC CLASSIFICATION ---")
print(
    "Difference <= 1e-12              :",
    roundtrip_level_631mer1
)
print(
    "Frozen decisions invariant       :",
    decision_invariant_631mer1
)
print(
    "Risk ranking invariant           :",
    ranking_invariant_631mer1
)
print(
    "SAFE SERIALIZATION DIFFERENCE    :",
    safe_serialization_difference_631mer1
)

print(
    "\nAudit SHA-256:",
    FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1
)

print(
    "\nPASS — diagnostic completed. "
    "No publication-strengthening metric was computed."
)

In [ ]:
# STEP 6.31M-E-R2 — RESUME PUBLICATION-STRENGTHENING ANALYSIS
#                    AFTER FD002 SERIALIZATION AUDIT
#
# PURPOSE
# -------
# Continue STEP 6.31M-E after 6.31M-E-R1 demonstrated that the
# decision/scoring reliability-risk difference is only float64/CSV
# serialization noise:
#
#   - max difference <= machine epsilon
#   - masks identical
#   - threshold decisions identical
#   - risk ranking identical
#
# NEW REPORTING-ONLY ANALYSES
# ---------------------------
# 1. Risk–coverage curves
# 2. MAE-AURC / normalized MAE-AURC
# 3. Matched random-rejection baseline at frozen coverage
# 4. Fixed-policy endpoint bootstrap
#
# NO:
#   - model inference
#   - MC dropout
#   - raw TEST/RUL access
#   - prediction regeneration
#   - label reconstruction
#   - threshold tuning
#   - policy change
#   - FD003 reconstruction
#   - FD004 reopening

import json
import hashlib
import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631mer2 = (
    "all_checks_passed_631l",
    "all_checks_passed_631md",
    "all_checks_passed_631mer1",
    "safe_serialization_difference_631mer1",
    "FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1",

    # Helpers/arrays created before the original 6.31M-E exception.
    "_risk_coverage_curve_631me",
    "_matched_random_rejection_631me",
    "_fixed_policy_bootstrap_631me",

    "fd001_error_631me",
    "fd001_accept_631me",
    "fd001_risk_631me",
    "fd001_id_631me",

    "fd002_error_631me",
    "fd002_accept_631me",
    "fd002_risk_631me",
    "fd002_id_631me",

    "FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB",
)

missing_631mer2 = [
    name
    for name in required_631mer2
    if name not in globals()
]

if missing_631mer2:
    raise RuntimeError(
        "STEP 6.31M-E-R2 missing prerequisite state: "
        + ", ".join(missing_631mer2)
    )

if not bool(all_checks_passed_631mer1):
    raise RuntimeError(
        "6.31M-E-R1 diagnostic did not pass."
    )

if not bool(
    safe_serialization_difference_631mer1
):
    raise RuntimeError(
        "FD002 risk discrepancy was not classified as "
        "safe serialization noise."
    )


# =============================================================================
# 1) HARD PROVENANCE GUARD
# =============================================================================

if not bool(
    accept_exact_equal_631mer1
):
    raise RuntimeError(
        "FD002 accept masks are not identical."
    )

if not bool(
    abstain_exact_equal_631mer1
):
    raise RuntimeError(
        "FD002 abstain masks are not identical."
    )

if not bool(
    score_rule_exact_631mer1
    and decision_rule_exact_631mer1
    and cross_rule_exact_631mer1
):
    raise RuntimeError(
        "FD002 frozen threshold decisions are not invariant."
    )

if not bool(
    risk_order_exact_631mer1
):
    raise RuntimeError(
        "FD002 risk ranking changed across serialized artifacts."
    )

# Explicit numerical safeguard:
# allow only a tiny number of float64 ulps.
max_allowed_serialization_error_631mer2 = (
    4.0
    * np.finfo(np.float64).eps
)

if (
    max_abs_risk_diff_631mer1
    >
    max_allowed_serialization_error_631mer2
):
    raise RuntimeError(
        "FD002 risk difference exceeds the allowed "
        "float64 serialization bound."
    )

# Ensure representation noise is nowhere near the decision boundary.
if not (
    max_abs_risk_diff_631mer1
    <
    minimum_threshold_margin_631mer1
):
    raise RuntimeError(
        "FD002 serialization difference could affect "
        "the frozen threshold decision."
    )


# =============================================================================
# 2) ROLE / SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MER2 = False
RAW_TEST_ACCESS_631MER2 = False
RAW_RUL_ACCESS_631MER2 = False
MODEL_INFERENCE_631MER2 = False
MC_DROPOUT_631MER2 = False
PREDICTION_REGENERATION_631MER2 = False
LABEL_RECONSTRUCTION_631MER2 = False
THRESHOLD_TUNING_631MER2 = False
POLICY_SELECTION_631MER2 = False
PRIMARY_RESULT_CHANGED_631MER2 = False

POSTHOC_REPORTING_ANALYSIS_631MER2 = True
RISK_COVERAGE_COMPUTED_631MER2 = True
AURC_COMPUTED_631MER2 = True
MATCHED_RANDOM_REJECTION_COMPUTED_631MER2 = True
FIXED_POLICY_BOOTSTRAP_COMPUTED_631MER2 = True

FD003_RECONSTRUCTED_631MER2 = False
FD003_RERUN_631MER2 = False
FD004_REOPENED_631MER2 = False


# =============================================================================
# 3) RISK–COVERAGE + AURC
#
# IMPORTANT:
# For FD002, fd002_risk_631me comes from the official scoring artifact.
# R1 proved that this representation has:
#   - identical ranking,
#   - identical accept/abstain decisions,
#   - only machine-epsilon differences
# relative to the frozen pre-label decision representation.
# =============================================================================

(
    FD001_PUBLICATION_RISK_COVERAGE_CURVE_631MER2,
    fd001_aurc_summary_631mer2,
) = _risk_coverage_curve_631me(
    "FD001_capped_RUL",
    fd001_id_631me,
    fd001_risk_631me,
    fd001_error_631me,
)


(
    FD002_PUBLICATION_RISK_COVERAGE_CURVE_631MER2,
    fd002_aurc_summary_631mer2,
) = _risk_coverage_curve_631me(
    "FD002_capped_RUL",
    fd002_id_631me,
    fd002_risk_631me,
    fd002_error_631me,
)


FD001_FD002_PUBLICATION_AURC_SUMMARY_631MER2 = (
    pd.DataFrame(
        [
            fd001_aurc_summary_631mer2,
            fd002_aurc_summary_631mer2,
        ]
    )
)


# =============================================================================
# 4) MATCHED RANDOM-REJECTION BASELINE
# =============================================================================

fd001_random_summary_631mer2 = (
    _matched_random_rejection_631me(
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        reps=20000,
        seed=631051,
    )
)


fd002_random_summary_631mer2 = (
    _matched_random_rejection_631me(
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        reps=20000,
        seed=631052,
    )
)


FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631MER2 = (
    pd.DataFrame(
        [
            fd001_random_summary_631mer2,
            fd002_random_summary_631mer2,
        ]
    )
)


# =============================================================================
# 5) FIXED-POLICY ENDPOINT BOOTSTRAP
# =============================================================================

fd001_bootstrap_summary_631mer2 = (
    _fixed_policy_bootstrap_631me(
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        reps=20000,
        seed=631061,
    )
)


fd002_bootstrap_summary_631mer2 = (
    _fixed_policy_bootstrap_631me(
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        reps=20000,
        seed=631062,
    )
)


FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631MER2 = (
    pd.DataFrame(
        [
            fd001_bootstrap_summary_631mer2,
            fd002_bootstrap_summary_631mer2,
        ]
    )
)


# =============================================================================
# 6) FD003 — PRESERVE ORIGINAL NEGATIVE HELD-OUT RESULT
# =============================================================================

FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2 = {
    "dataset":
        "FD003",

    "new_row_level_posthoc_analysis_performed":
        False,

    "reason":
        (
            "authoritative held-out row-level arrays were not "
            "persisted/recoverable without reconstruction or rerun"
        ),

    "original_frozen_result_preserved":
        True,

    "endpoint_count":
        int(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "endpoint_count"
            ]
        ),

    "realized_coverage":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "realized_coverage"
            ]
        ),

    "full_mae":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "full_mae"
            ]
        ),

    "accepted_mae":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "accepted_mae"
            ]
        ),

    "selective_mae_gain":
        float(
            FD003_HELDOUT_TEST_SCORE_CONTRACT_629BB[
                "selective_mae_gain"
            ]
        ),

    "interpretation":
        (
            "original frozen held-out negative result retained; "
            "no post-hoc reconstruction performed"
        ),
}


# =============================================================================
# 7) COMPACT PUBLICATION SUMMARY
# =============================================================================

publication_rows_631mer2 = []

for (
    dataset,
    error,
    accept,
    aurc_summary,
    random_summary,
    bootstrap_summary,
) in (
    (
        "FD001_capped_RUL",
        fd001_error_631me,
        fd001_accept_631me,
        fd001_aurc_summary_631mer2,
        fd001_random_summary_631mer2,
        fd001_bootstrap_summary_631mer2,
    ),
    (
        "FD002_capped_RUL",
        fd002_error_631me,
        fd002_accept_631me,
        fd002_aurc_summary_631mer2,
        fd002_random_summary_631mer2,
        fd002_bootstrap_summary_631mer2,
    ),
):

    full_mae_631mer2 = float(
        np.mean(error)
    )

    accepted_mae_631mer2 = float(
        np.mean(
            error[accept]
        )
    )

    publication_rows_631mer2.append(
        {
            "dataset":
                dataset,

            "n":
                int(len(error)),

            "frozen_accepted_n":
                int(
                    np.sum(accept)
                ),

            "frozen_coverage":
                float(
                    np.mean(accept)
                ),

            "full_mae":
                full_mae_631mer2,

            "frozen_accepted_mae":
                accepted_mae_631mer2,

            "frozen_mae_gain":
                float(
                    full_mae_631mer2
                    - accepted_mae_631mer2
                ),

            "normalized_mae_aurc":
                float(
                    aurc_summary[
                        "normalized_mae_aurc"
                    ]
                ),

            "aurc_relative_improvement_vs_random":
                float(
                    aurc_summary[
                        "relative_aurc_improvement_vs_random"
                    ]
                ),

            "matched_random_median_mae":
                float(
                    random_summary[
                        "random_mae_median"
                    ]
                ),

            "matched_random_median_minus_observed":
                float(
                    random_summary[
                        "random_median_minus_observed"
                    ]
                ),

            "random_fraction_as_good_or_better":
                float(
                    random_summary[
                        "fraction_random_as_good_or_better"
                    ]
                ),

            "bootstrap_gain_ci_low":
                float(
                    bootstrap_summary[
                        "selective_gain_ci_2_5"
                    ]
                ),

            "bootstrap_gain_ci_high":
                float(
                    bootstrap_summary[
                        "selective_gain_ci_97_5"
                    ]
                ),

            "bootstrap_fraction_gain_positive":
                float(
                    bootstrap_summary[
                        "fraction_selective_gain_positive"
                    ]
                ),

            "bootstrap_separation_ci_low":
                float(
                    bootstrap_summary[
                        "separation_ci_2_5"
                    ]
                ),

            "bootstrap_separation_ci_high":
                float(
                    bootstrap_summary[
                        "separation_ci_97_5"
                    ]
                ),

            "bootstrap_fraction_abstained_mae_higher":
                float(
                    bootstrap_summary[
                        "fraction_abstained_mae_higher"
                    ]
                ),
        }
    )


FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631MER2 = (
    pd.DataFrame(
        publication_rows_631mer2
    )
)


# =============================================================================
# 8) REPORTING CONTRACT
# =============================================================================

FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631MER2 = {
    "stage":
        "6.31M-E-R2",

    "role":
        "posthoc_reporting_only_publication_strengthening",

    "empirical_closure_preserved":
        True,

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "primary_result_changed":
        False,

    "fd002_serialization_audit_sha256":
        str(
            FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1
        ),

    "fd002_risk_representation_note":
        (
            "official scoring CSV risk used for post-hoc ranking; "
            "difference from pre-label decision CSV is bounded by "
            "one float64 machine epsilon in observed data, with "
            "identical risk ordering and identical frozen decisions"
        ),

    "fd002_max_abs_risk_serialization_difference":
        float(
            max_abs_risk_diff_631mer1
        ),

    "fd002_frozen_decision_invariant":
        True,

    "fd002_risk_ranking_invariant":
        True,

    "fd001_primary_target":
        "RUL_cap_125",

    "fd002_primary_target":
        "piecewise-capped RUL, cap=125",

    "risk_coverage_definition":
        (
            "ascending frozen reliability risk; "
            "outcome-independent endpoint-ID tie break"
        ),

    "aurc_definition":
        (
            "mean retained-subset MAE across retained sizes "
            "k=1,...,N"
        ),

    "matched_random_baseline":
        (
            "Monte-Carlo random subsets with exactly the frozen "
            "accepted count"
        ),

    "bootstrap_scope":
        (
            "held-out endpoint sampling variability conditional on "
            "the frozen trained model and frozen operating policy"
        ),

    "bootstrap_training_variability":
        False,

    "formal_training_significance_claim":
        False,

    "fd003_new_posthoc_analysis":
        False,

    "fd003_original_negative_result_preserved":
        True,

    "fd004_reopened":
        False,
}


def _json_sha_631mer2(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2 = (
    _json_sha_631mer2(
        FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631MER2
    )
)


all_checks_passed_631mer2 = True


# =============================================================================
# 9) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-E-R2 — PUBLICATION-STRENGTHENING "
    "SELECTIVE ANALYSIS (FD001 + FD002)"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Model inference / MC dropout      :", False)
print("Threshold/policy selection        :", False)
print("Primary held-out results changed  :", False)
print("FD002 serialization audit passed  :", True)
print("Post-hoc reporting analysis       :", True)
print("Risk–coverage / AURC              :", True)
print("Matched random rejection          :", True)
print("Fixed-policy bootstrap            :", True)
print("FD003 reconstructed/rerun         :", False)
print("FD004 reopened                    :", False)
print("6.31L empirical closure preserved :", True)


print(
    "\n--- PUBLICATION-STRENGTHENING SUMMARY ---"
)

print(
    FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631MER2.to_string(
        index=False
    )
)


print(
    "\n--- AURC DETAIL ---"
)

print(
    FD001_FD002_PUBLICATION_AURC_SUMMARY_631MER2.to_string(
        index=False
    )
)


print(
    "\n--- MATCHED RANDOM-REJECTION BASELINE ---"
)

print(
    FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631MER2.to_string(
        index=False
    )
)


print(
    "\n--- FIXED-POLICY BOOTSTRAP ---"
)

print(
    FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631MER2.to_string(
        index=False
    )
)


print(
    "\n--- FD003 PRESERVATION STATUS ---"
)

print(
    json.dumps(
        FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2,
        indent=2,
    )
)


print(
    "\nPublication-analysis contract SHA-256:",
    FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2,
)

print(
    "All checks passed                     :",
    all_checks_passed_631mer2,
)

print(
    "\nPASS — publication-strengthening analyses completed "
    "without modifying any frozen scientific result."
)

print(
    "Results are still in-memory only. Do not persist them until "
    "their scientific interpretation has been reviewed."
)

In [ ]:
# STEP 6.31M-E-R3 — RISK-TIE SENSITIVITY AUDIT
#
# PURPOSE
# -------
# Determine whether the publication-strengthening AURC conclusions are
# sensitive to arbitrary ordering among endpoints having identical
# reliability-risk values.
#
# IMPORTANT:
# - no model execution
# - no TEST/RUL raw access
# - no prediction regeneration
# - no threshold change
# - no policy selection
# - no primary-result change
#
# This is reporting-only sensitivity analysis.
#
# Two outputs:
#   1. threshold-attainable risk–coverage curve:
#      all endpoints sharing the same risk enter simultaneously;
#
#   2. random within-tie permutation sensitivity for standard discrete AURC.
#
# The second asks:
#   "If tied endpoints are ordered differently, how much can AURC change?"

import hashlib
import json
import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631mer3 = (
    "all_checks_passed_631mer2",
    "FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2",
    "fd001_risk_631me",
    "fd001_error_631me",
    "fd001_id_631me",
    "fd002_risk_631me",
    "fd002_error_631me",
    "fd002_id_631me",
    "fd001_aurc_summary_631mer2",
    "fd002_aurc_summary_631mer2",
)

missing_631mer3 = [
    x for x in required_631mer3
    if x not in globals()
]

if missing_631mer3:
    raise RuntimeError(
        "STEP 6.31M-E-R3 missing prerequisite state: "
        + ", ".join(missing_631mer3)
    )

if not bool(all_checks_passed_631mer2):
    raise RuntimeError(
        "6.31M-E-R2 did not complete successfully."
    )


# =============================================================================
# 1) SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MER3 = False
RAW_TEST_ACCESS_631MER3 = False
RAW_RUL_ACCESS_631MER3 = False
MODEL_INFERENCE_631MER3 = False
MC_DROPOUT_631MER3 = False
PREDICTION_REGENERATION_631MER3 = False
LABEL_RECONSTRUCTION_631MER3 = False
THRESHOLD_TUNING_631MER3 = False
POLICY_SELECTION_631MER3 = False
PRIMARY_RESULT_CHANGED_631MER3 = False

POSTHOC_TIE_SENSITIVITY_ONLY_631MER3 = True
FD003_REOPENED_631MER3 = False
FD004_REOPENED_631MER3 = False


# =============================================================================
# 2) THRESHOLD-ATTAINABLE CURVE
#
# All endpoints with identical risk enter at the same threshold.
# Therefore this curve has no arbitrary within-tie ordering.
# =============================================================================

def _threshold_attainable_curve_631mer3(
    dataset,
    risk,
    abs_error,
):
    risk = np.asarray(
        risk,
        dtype=np.float64,
    )

    abs_error = np.asarray(
        abs_error,
        dtype=np.float64,
    )

    if len(risk) != len(abs_error):
        raise RuntimeError(
            f"{dataset}: vector length mismatch."
        )

    n = len(risk)

    order = np.argsort(
        risk,
        kind="mergesort",
    )

    risk_s = risk[order]
    err_s = abs_error[order]

    cumulative_error = np.cumsum(
        err_s,
        dtype=np.float64,
    )

    # End position of each exact risk group.
    group_end = np.r_[
        np.flatnonzero(
            risk_s[1:]
            != risk_s[:-1]
        ),
        n - 1,
    ]

    retained_n = (
        group_end + 1
    ).astype(int)

    coverage = (
        retained_n
        / float(n)
    )

    selective_mae = (
        cumulative_error[group_end]
        / retained_n
    )

    risk_threshold = (
        risk_s[group_end]
    )

    group_start = np.r_[
        0,
        group_end[:-1] + 1,
    ]

    group_size = (
        group_end
        - group_start
        + 1
    ).astype(int)

    full_mae = float(
        abs_error.mean()
    )

    curve = pd.DataFrame(
        {
            "dataset":
                dataset,

            "risk_threshold":
                risk_threshold,

            "risk_group_size":
                group_size,

            "retained_n":
                retained_n,

            "coverage":
                coverage,

            "selective_mae":
                selective_mae,

            "full_mae":
                full_mae,

            "mae_gain":
                full_mae
                - selective_mae,
        }
    )

    return curve


# =============================================================================
# 3) STANDARD DISCRETE AURC UNDER RANDOM WITHIN-TIE PERMUTATIONS
# =============================================================================

def _tie_permutation_aurc_631mer3(
    dataset,
    risk,
    abs_error,
    *,
    reps=20000,
    seed=631071,
):
    risk = np.asarray(
        risk,
        dtype=np.float64,
    )

    abs_error = np.asarray(
        abs_error,
        dtype=np.float64,
    )

    n = len(risk)

    if len(abs_error) != n:
        raise RuntimeError(
            f"{dataset}: vector length mismatch."
        )

    unique_risk = np.unique(
        risk
    )

    groups = [
        np.flatnonzero(
            risk == value
        )
        for value in np.sort(
            unique_risk
        )
    ]

    rng = np.random.default_rng(
        int(seed)
    )

    aurc_values = np.empty(
        int(reps),
        dtype=np.float64,
    )

    for rep in range(
        int(reps)
    ):
        pieces = []

        for group in groups:

            if len(group) <= 1:
                pieces.append(group)

            else:
                pieces.append(
                    rng.permutation(
                        group
                    )
                )

        order = np.concatenate(
            pieces
        )

        err_s = abs_error[
            order
        ]

        selective_mae = (
            np.cumsum(
                err_s,
                dtype=np.float64,
            )
            /
            np.arange(
                1,
                n + 1,
                dtype=np.float64,
            )
        )

        aurc_values[rep] = float(
            selective_mae.mean()
        )

    full_mae = float(
        abs_error.mean()
    )

    normalized = (
        aurc_values
        / full_mae
    )

    return {
        "dataset":
            dataset,

        "n":
            int(n),

        "unique_risk_values":
            int(len(unique_risk)),

        "tied_endpoint_excess":
            int(
                n
                - len(unique_risk)
            ),

        "tie_permutations":
            int(reps),

        "seed":
            int(seed),

        "full_mae":
            full_mae,

        "aurc_mean":
            float(
                aurc_values.mean()
            ),

        "aurc_median":
            float(
                np.median(
                    aurc_values
                )
            ),

        "aurc_ci_2_5":
            float(
                np.quantile(
                    aurc_values,
                    0.025,
                )
            ),

        "aurc_ci_97_5":
            float(
                np.quantile(
                    aurc_values,
                    0.975,
                )
            ),

        "normalized_aurc_mean":
            float(
                normalized.mean()
            ),

        "normalized_aurc_median":
            float(
                np.median(
                    normalized
                )
            ),

        "normalized_aurc_ci_2_5":
            float(
                np.quantile(
                    normalized,
                    0.025,
                )
            ),

        "normalized_aurc_ci_97_5":
            float(
                np.quantile(
                    normalized,
                    0.975,
                )
            ),

        "fraction_normalized_aurc_below_1":
            float(
                np.mean(
                    normalized < 1.0
                )
            ),

        "minimum_normalized_aurc":
            float(
                normalized.min()
            ),

        "maximum_normalized_aurc":
            float(
                normalized.max()
            ),
    }


# =============================================================================
# 4) COMPUTE TIE-INVARIANT THRESHOLD CURVES
# =============================================================================

FD001_THRESHOLD_ATTAINABLE_CURVE_631MER3 = (
    _threshold_attainable_curve_631mer3(
        "FD001_capped_RUL",
        fd001_risk_631me,
        fd001_error_631me,
    )
)

FD002_THRESHOLD_ATTAINABLE_CURVE_631MER3 = (
    _threshold_attainable_curve_631mer3(
        "FD002_capped_RUL",
        fd002_risk_631me,
        fd002_error_631me,
    )
)


# =============================================================================
# 5) COMPUTE WITHIN-TIE AURC SENSITIVITY
# =============================================================================

fd001_tie_summary_631mer3 = (
    _tie_permutation_aurc_631mer3(
        "FD001_capped_RUL",
        fd001_risk_631me,
        fd001_error_631me,
        reps=20000,
        seed=631071,
    )
)

fd002_tie_summary_631mer3 = (
    _tie_permutation_aurc_631mer3(
        "FD002_capped_RUL",
        fd002_risk_631me,
        fd002_error_631me,
        reps=20000,
        seed=631072,
    )
)


# Add deterministic endpoint-ID AURC from R2 for direct comparison.

fd001_tie_summary_631mer3[
    "deterministic_endpoint_id_normalized_aurc"
] = float(
    fd001_aurc_summary_631mer2[
        "normalized_mae_aurc"
    ]
)

fd002_tie_summary_631mer3[
    "deterministic_endpoint_id_normalized_aurc"
] = float(
    fd002_aurc_summary_631mer2[
        "normalized_mae_aurc"
    ]
)


FD001_FD002_TIE_SENSITIVITY_SUMMARY_631MER3 = (
    pd.DataFrame(
        [
            fd001_tie_summary_631mer3,
            fd002_tie_summary_631mer3,
        ]
    )
)


# =============================================================================
# 6) PUBLICATION-SAFETY CLASSIFICATION
#
# This is NOT hypothesis testing.
#
# "robustly_better_than_random_ranking" means all observed
# within-tie permutation normalized AURCs remained < 1.
# =============================================================================

tie_publication_assessment_631mer3 = {}

for row in (
    fd001_tie_summary_631mer3,
    fd002_tie_summary_631mer3,
):

    tie_publication_assessment_631mer3[
        row["dataset"]
    ] = {
        "all_sampled_tie_orders_below_random_reference":
            bool(
                row[
                    "maximum_normalized_aurc"
                ]
                < 1.0
            ),

        "fraction_sampled_tie_orders_below_random_reference":
            float(
                row[
                    "fraction_normalized_aurc_below_1"
                ]
            ),

        "tie_sensitivity_range":
            float(
                row[
                    "maximum_normalized_aurc"
                ]
                -
                row[
                    "minimum_normalized_aurc"
                ]
            ),
    }


# =============================================================================
# 7) CONTRACT
# =============================================================================

FD001_FD002_TIE_SENSITIVITY_CONTRACT_631MER3 = {
    "stage":
        "6.31M-E-R3",

    "upstream_publication_analysis_sha256":
        str(
            FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2
        ),

    "role":
        "posthoc_reporting_only_risk_tie_sensitivity",

    "scientific_recomputation":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "primary_result_changed":
        False,

    "tie_handling":
        (
            "exact equal-risk groups preserved; only ordering "
            "within equal-risk groups randomized"
        ),

    "tie_permutation_replicates":
        20000,

    "random_reference":
        (
            "normalized MAE-AURC = 1 corresponds to expected "
            "random-ranking selective MAE"
        ),

    "fd003_reopened":
        False,

    "fd004_reopened":
        False,
}


def _json_sha_631mer3(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        ).encode("utf-8")
    ).hexdigest()


FD001_FD002_TIE_SENSITIVITY_CONTRACT_SHA256_631MER3 = (
    _json_sha_631mer3(
        FD001_FD002_TIE_SENSITIVITY_CONTRACT_631MER3
    )
)


all_checks_passed_631mer3 = True


# =============================================================================
# 8) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-E-R3 — RISK-TIE SENSITIVITY AUDIT"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Model inference / MC dropout      :", False)
print("Prediction regeneration           :", False)
print("Threshold/policy selection        :", False)
print("Primary results changed           :", False)
print("Post-hoc tie sensitivity only     :", True)


print(
    "\n--- TIE-SENSITIVITY SUMMARY ---"
)

print(
    FD001_FD002_TIE_SENSITIVITY_SUMMARY_631MER3.to_string(
        index=False
    )
)


print(
    "\n--- PUBLICATION-SAFETY ASSESSMENT ---"
)

print(
    json.dumps(
        tie_publication_assessment_631mer3,
        indent=2,
    )
)


print(
    "\n--- THRESHOLD-ATTAINABLE CURVE SIZES ---"
)

print(
    "FD001 threshold points:",
    len(
        FD001_THRESHOLD_ATTAINABLE_CURVE_631MER3
    )
)

print(
    "FD002 threshold points:",
    len(
        FD002_THRESHOLD_ATTAINABLE_CURVE_631MER3
    )
)


print(
    "\nTie-sensitivity contract SHA-256:",
    FD001_FD002_TIE_SENSITIVITY_CONTRACT_SHA256_631MER3,
)

print(
    "All checks passed:",
    all_checks_passed_631mer3,
)

print(
    "\nPASS — AURC tie sensitivity evaluated without altering "
    "the frozen empirical results."
)

print(
    "Do not persist yet; inspect the tie-sensitivity result first."
)

In [ ]:
# STEP 6.31M-F — FREEZE PUBLICATION-STRENGTHENING REPORTING EVIDENCE
#
# PURPOSE
# -------
# Persist the already-reviewed reporting-only strengthening analyses from:
#
#   6.31M-E-R1  FD002 serialization/identity audit
#   6.31M-E-R2  risk–coverage, AURC, random rejection, bootstrap
#   6.31M-E-R3  risk-tie sensitivity
#
# THIS STEP DOES NOT COMPUTE NEW SCIENCE.
#
# It creates a separate manuscript-ready strengthening bundle without
# modifying/replacing the authoritative 6.31L closure artifacts.
#
# NO:
#   - model inference
#   - MC dropout
#   - raw TEST/RUL access
#   - prediction regeneration
#   - label reconstruction
#   - threshold tuning
#   - policy selection
#   - new analysis
#   - FD003 reconstruction
#   - FD004 reopening

from pathlib import Path
import json
import hashlib
import os

import numpy as np
import pandas as pd


# =============================================================================
# 0) PRECONDITIONS
# =============================================================================

required_631mf = (
    "all_checks_passed_631l",
    "all_checks_passed_631mer1",
    "all_checks_passed_631mer2",
    "all_checks_passed_631mer3",

    "FD002_RISK_SERIALIZATION_AUDIT_631MER1",
    "FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1",

    "FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631MER2",
    "FD001_FD002_PUBLICATION_AURC_SUMMARY_631MER2",
    "FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631MER2",
    "FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631MER2",

    "FD001_PUBLICATION_RISK_COVERAGE_CURVE_631MER2",
    "FD002_PUBLICATION_RISK_COVERAGE_CURVE_631MER2",

    "FD001_THRESHOLD_ATTAINABLE_CURVE_631MER3",
    "FD002_THRESHOLD_ATTAINABLE_CURVE_631MER3",
    "FD001_FD002_TIE_SENSITIVITY_SUMMARY_631MER3",

    "FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2",

    "FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631MER2",
    "FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2",

    "FD001_FD002_TIE_SENSITIVITY_CONTRACT_631MER3",
    "FD001_FD002_TIE_SENSITIVITY_CONTRACT_SHA256_631MER3",

    "FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L",
)

missing_631mf = [
    name
    for name in required_631mf
    if name not in globals()
]

if missing_631mf:
    raise RuntimeError(
        "STEP 6.31M-F missing prerequisite state: "
        + ", ".join(missing_631mf)
    )

if not (
    bool(all_checks_passed_631l)
    and bool(all_checks_passed_631mer1)
    and bool(all_checks_passed_631mer2)
    and bool(all_checks_passed_631mer3)
):
    raise RuntimeError(
        "One or more upstream closure/reporting audits are not PASS."
    )


# =============================================================================
# 1) SAFETY DECLARATIONS
# =============================================================================

SCIENTIFIC_RECOMPUTATION_631MF = False
NEW_ANALYSIS_631MF = False
RAW_TEST_ACCESS_631MF = False
RAW_RUL_ACCESS_631MF = False
MODEL_INFERENCE_631MF = False
MC_DROPOUT_631MF = False
PREDICTION_REGENERATION_631MF = False
LABEL_RECONSTRUCTION_631MF = False
THRESHOLD_TUNING_631MF = False
POLICY_SELECTION_631MF = False
PRIMARY_RESULT_CHANGED_631MF = False
FD003_RECONSTRUCTED_631MF = False
FD004_REOPENED_631MF = False

REPORTING_ARTIFACT_PERSISTENCE_ONLY_631MF = True


# =============================================================================
# 2) OUTPUT ROOT
# =============================================================================

output_root_631mf = Path(
    r"C:\RUL_Reliability_Research\06_results"
    r"\cross_dataset_synthesis\manuscript_ready"
    r"\publication_strengthening_631mf"
)

checkpoint_root_631mf = Path(
    r"C:\RUL_Reliability_Research\checkpoints"
)

output_root_631mf.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_root_631mf.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3) SERIALIZATION HELPERS
# =============================================================================

def _sha_bytes_631mf(data):
    return hashlib.sha256(data).hexdigest()


def _sha_file_631mf(path, chunk=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def _jsonable_631mf(obj):

    if isinstance(
        obj,
        (
            np.integer,
            np.floating,
        ),
    ):
        return obj.item()

    if isinstance(
        obj,
        np.ndarray,
    ):
        return obj.tolist()

    if isinstance(
        obj,
        Path,
    ):
        return str(obj)

    raise TypeError(
        f"Not JSON serializable: {type(obj).__name__}"
    )


def _json_bytes_631mf(obj):
    return (
        json.dumps(
            obj,
            sort_keys=True,
            indent=2,
            ensure_ascii=False,
            default=_jsonable_631mf,
        )
        + "\n"
    ).encode("utf-8")


def _csv_bytes_631mf(df):
    text = df.to_csv(
        index=False,
        float_format="%.17g",
        lineterminator="\n",
    )

    return text.encode("utf-8")


def _write_once_631mf(path, data):
    """
    Never silently overwrite a differing frozen artifact.

    Re-running the cell is allowed only when bytes are identical.
    """

    path = Path(path)

    if path.exists():

        existing = path.read_bytes()

        if existing != data:
            raise RuntimeError(
                "Refusing to overwrite non-identical frozen artifact: "
                + str(path)
            )

        return {
            "path": str(path),
            "sha256": _sha_bytes_631mf(existing),
            "size_bytes": len(existing),
            "write_status": "already_present_identical",
        }

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_bytes(data)

    os.replace(
        tmp,
        path,
    )

    written = path.read_bytes()

    if written != data:
        raise RuntimeError(
            "Post-write byte verification failed: "
            + str(path)
        )

    return {
        "path": str(path),
        "sha256": _sha_bytes_631mf(written),
        "size_bytes": len(written),
        "write_status": "written_and_verified",
    }


# =============================================================================
# 4) SCIENTIFIC INTERPRETATION LOCK
#
# These statements intentionally distinguish:
#
#   - frozen primary results;
#   - matched-random reporting evidence;
#   - endpoint-bootstrap sampling variability;
#   - risk-tie sensitivity;
#   - training variability (NOT measured).
# =============================================================================

PUBLICATION_STRENGTHENING_INTERPRETATION_631MF = {

    "stage":
        "6.31M-F",

    "overall_role":
        "posthoc_reporting_only_publication_strengthening",

    "fd001": {
        "status":
            "supportive_but_statistically_cautious",

        "primary_target":
            "RUL_cap_125",

        "interpretation":
            (
                "The frozen FD001 selective policy produced a favorable "
                "held-out point estimate and meaningful matched-random "
                "separation. Endpoint-bootstrap directional stability was "
                "high, but the 95% percentile interval for selective MAE "
                "gain included zero. Risk-ranking advantage remained below "
                "the random-ranking nAURC reference across all sampled "
                "within-tie orderings, although the precise AURC magnitude "
                "is sensitive to the large number of tied risk values."
            ),

        "claim_boundary": (
            "Report as supportive/directional evidence; do not claim a "
            "95%-level statistically conclusive selective improvement."
        ),

        "aurc_reporting_guidance": (
            "Do not present the single endpoint-ID-tiebroken FD001 AURC as "
            "a uniquely defined quantity. Prefer the tie-sensitivity "
            "distribution and/or threshold-attainable curve, noting that "
            "all 20,000 sampled tie orders remained better than the "
            "random-ranking reference."
        ),
    },

    "fd002": {
        "status":
            "strong_and_consistent",

        "primary_target":
            "piecewise-capped RUL, cap=125",

        "interpretation":
            (
                "The frozen FD002 reliability policy shows strong held-out "
                "error separation. The uncertainty-selected accepted subset "
                "outperformed matched random rejection at the same coverage, "
                "the endpoint-bootstrap selective-gain interval remained "
                "strictly positive, and the nAURC ranking advantage was "
                "essentially insensitive to tied-risk ordering."
            ),

        "claim_boundary": (
            "Strong held-out evidence conditional on the frozen trained "
            "model/policy; bootstrap results do not quantify retraining or "
            "random-seed variability."
        ),
    },

    "fd003": {
        "status":
            "heldout_negative_counterexample",

        "interpretation":
            (
                "The original frozen held-out negative result remains "
                "authoritative. No row-level post-hoc reconstruction or "
                "new selective analysis was performed."
            ),
    },

    "fd004": {
        "status":
            "structural_protocol_guard_unscored",

        "interpretation":
            (
                "Held-out TEST remains unscored under the frozen fixed-window "
                "protocol because short TEST trajectories violated the "
                "original evaluation admissibility contract."
            ),
    },

    "global_claim": (
        "Reliability-aware selective prediction produced useful error "
        "separation in FD001 and especially FD002, but the benefit was not "
        "universal: FD003 remained a held-out negative counterexample and "
        "FD004 remained structurally unscored under the frozen protocol."
    ),

    "prohibited_claims": [
        "universal selective-prediction improvement",
        "formal model-training significance from endpoint bootstrap",
        "a uniquely defined FD001 AURC despite tied risk values",
        "FD004 held-out TEST performance",
        "reconstructed FD003 row-level post-hoc results",
        "post-TEST policy optimization",
    ],
}


# =============================================================================
# 5) BUILD MANUSCRIPT-READY TEXT SUMMARY
# =============================================================================

summary_df_631mf = (
    FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631MER2
)

tie_df_631mf = (
    FD001_FD002_TIE_SENSITIVITY_SUMMARY_631MER3
)

fd001_summary_631mf = (
    summary_df_631mf[
        summary_df_631mf["dataset"]
        == "FD001_capped_RUL"
    ]
    .iloc[0]
)

fd002_summary_631mf = (
    summary_df_631mf[
        summary_df_631mf["dataset"]
        == "FD002_capped_RUL"
    ]
    .iloc[0]
)

fd001_tie_631mf = (
    tie_df_631mf[
        tie_df_631mf["dataset"]
        == "FD001_capped_RUL"
    ]
    .iloc[0]
)

fd002_tie_631mf = (
    tie_df_631mf[
        tie_df_631mf["dataset"]
        == "FD002_capped_RUL"
    ]
    .iloc[0]
)


markdown_631mf = f"""# Publication-Strengthening Reporting Evidence — STEP 6.31M-F

## Governance

This bundle is reporting-only.

- Scientific recomputation: False
- Model inference / MC dropout: False
- Raw TEST access: False
- Raw RUL access: False
- Prediction regeneration: False
- Label reconstruction: False
- Threshold or policy selection: False
- Primary held-out results changed: False
- FD003 reconstructed: False
- FD004 reopened: False

The original STEP 6.31L empirical closure remains authoritative.

## FD001 — capped RUL primary target

Frozen TEST coverage: {fd001_summary_631mf['frozen_coverage']:.6f}

Full MAE: {fd001_summary_631mf['full_mae']:.6f}

Accepted MAE: {fd001_summary_631mf['frozen_accepted_mae']:.6f}

Frozen selective MAE gain: {fd001_summary_631mf['frozen_mae_gain']:.6f}

Matched-random fraction as good or better:
{fd001_summary_631mf['random_fraction_as_good_or_better']:.6f}

Endpoint-bootstrap selective-gain 95% percentile interval:
[{fd001_summary_631mf['bootstrap_gain_ci_low']:.6f},
 {fd001_summary_631mf['bootstrap_gain_ci_high']:.6f}]

Fraction of endpoint-bootstrap replicates with positive selective gain:
{fd001_summary_631mf['bootstrap_fraction_gain_positive']:.6f}

FD001 contains {int(fd001_tie_631mf['unique_risk_values'])} unique risk values
across {int(fd001_tie_631mf['n'])} endpoints.

Within-tie permutation normalized MAE-AURC:
- mean: {fd001_tie_631mf['normalized_aurc_mean']:.6f}
- 2.5%: {fd001_tie_631mf['normalized_aurc_ci_2_5']:.6f}
- 97.5%: {fd001_tie_631mf['normalized_aurc_ci_97_5']:.6f}
- minimum sampled: {fd001_tie_631mf['minimum_normalized_aurc']:.6f}
- maximum sampled: {fd001_tie_631mf['maximum_normalized_aurc']:.6f}
- fraction below random-ranking reference 1.0:
  {fd001_tie_631mf['fraction_normalized_aurc_below_1']:.6f}

Interpretation:
FD001 provides favorable and directionally stable evidence, but its
endpoint-bootstrap 95% interval includes zero. The direction of the AURC
advantage is robust to tied-risk ordering, while its exact magnitude is
tie-sensitive.

## FD002 — capped RUL

Frozen TEST coverage: {fd002_summary_631mf['frozen_coverage']:.6f}

Full MAE: {fd002_summary_631mf['full_mae']:.6f}

Accepted MAE: {fd002_summary_631mf['frozen_accepted_mae']:.6f}

Frozen selective MAE gain: {fd002_summary_631mf['frozen_mae_gain']:.6f}

Matched-random fraction as good or better:
{fd002_summary_631mf['random_fraction_as_good_or_better']:.6f}

Endpoint-bootstrap selective-gain 95% percentile interval:
[{fd002_summary_631mf['bootstrap_gain_ci_low']:.6f},
 {fd002_summary_631mf['bootstrap_gain_ci_high']:.6f}]

Fraction of endpoint-bootstrap replicates with positive selective gain:
{fd002_summary_631mf['bootstrap_fraction_gain_positive']:.6f}

Within-tie permutation normalized MAE-AURC:
- mean: {fd002_tie_631mf['normalized_aurc_mean']:.6f}
- 2.5%: {fd002_tie_631mf['normalized_aurc_ci_2_5']:.6f}
- 97.5%: {fd002_tie_631mf['normalized_aurc_ci_97_5']:.6f}
- minimum sampled: {fd002_tie_631mf['minimum_normalized_aurc']:.6f}
- maximum sampled: {fd002_tie_631mf['maximum_normalized_aurc']:.6f}

Interpretation:
FD002 provides strong and consistent evidence that the frozen reliability
ranking identifies a lower-error accepted subset. The result is robust to
matched random rejection, endpoint resampling, and tied-risk ordering.

## FD003

No new row-level analysis was performed.

Original frozen held-out result:
- coverage: {FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2['realized_coverage']:.6f}
- full MAE: {FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2['full_mae']:.6f}
- accepted MAE: {FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2['accepted_mae']:.6f}
- selective MAE gain: {FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2['selective_mae_gain']:.6f}

Interpretation:
The held-out negative result remains unchanged and is retained as an explicit
counterexample to universal selective-prediction improvement.

## FD004

No new held-out analysis was performed.

FD004 remains:
NOT_SCORED_UNDER_FROZEN_PROTOCOL_DUE_TO_SHORT_TEST_TRAJECTORIES

## Statistical boundary

Matched-random rejection is a reporting baseline at exactly matched frozen
coverage.

Endpoint bootstrap quantifies sampling variability conditional on the already
frozen trained model and policy.

It does not quantify retraining variability, random-seed variability, or
architecture-selection uncertainty.

No formal model-training significance claim is made.
"""


# =============================================================================
# 6) PERSIST ARTIFACTS
# =============================================================================

artifacts_631mf = {}


def _save_csv_631mf(filename, df):

    path = (
        output_root_631mf
        / filename
    )

    artifacts_631mf[
        filename
    ] = _write_once_631mf(
        path,
        _csv_bytes_631mf(df),
    )


def _save_json_631mf(filename, obj):

    path = (
        output_root_631mf
        / filename
    )

    artifacts_631mf[
        filename
    ] = _write_once_631mf(
        path,
        _json_bytes_631mf(obj),
    )


_save_csv_631mf(
    "publication_strengthening_summary_631mf.csv",
    FD001_FD002_PUBLICATION_STRENGTHENING_SUMMARY_631MER2,
)

_save_csv_631mf(
    "aurc_summary_631mf.csv",
    FD001_FD002_PUBLICATION_AURC_SUMMARY_631MER2,
)

_save_csv_631mf(
    "matched_random_rejection_summary_631mf.csv",
    FD001_FD002_MATCHED_RANDOM_REJECTION_SUMMARY_631MER2,
)

_save_csv_631mf(
    "fixed_policy_bootstrap_summary_631mf.csv",
    FD001_FD002_FIXED_POLICY_BOOTSTRAP_SUMMARY_631MER2,
)

_save_csv_631mf(
    "tie_sensitivity_summary_631mf.csv",
    FD001_FD002_TIE_SENSITIVITY_SUMMARY_631MER3,
)

_save_csv_631mf(
    "fd001_risk_coverage_curve_631mf.csv",
    FD001_PUBLICATION_RISK_COVERAGE_CURVE_631MER2,
)

_save_csv_631mf(
    "fd002_risk_coverage_curve_631mf.csv",
    FD002_PUBLICATION_RISK_COVERAGE_CURVE_631MER2,
)

_save_csv_631mf(
    "fd001_threshold_attainable_curve_631mf.csv",
    FD001_THRESHOLD_ATTAINABLE_CURVE_631MER3,
)

_save_csv_631mf(
    "fd002_threshold_attainable_curve_631mf.csv",
    FD002_THRESHOLD_ATTAINABLE_CURVE_631MER3,
)

_save_json_631mf(
    "fd002_serialization_audit_631mf.json",
    FD002_RISK_SERIALIZATION_AUDIT_631MER1,
)

_save_json_631mf(
    "fd003_preservation_status_631mf.json",
    FD003_PUBLICATION_STRENGTHENING_STATUS_631MER2,
)

_save_json_631mf(
    "publication_strengthening_interpretation_631mf.json",
    PUBLICATION_STRENGTHENING_INTERPRETATION_631MF,
)

_save_json_631mf(
    "publication_analysis_contract_631mf.json",
    FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_631MER2,
)

_save_json_631mf(
    "tie_sensitivity_contract_631mf.json",
    FD001_FD002_TIE_SENSITIVITY_CONTRACT_631MER3,
)


markdown_path_631mf = (
    output_root_631mf
    / "publication_strengthening_summary_631mf.md"
)

artifacts_631mf[
    markdown_path_631mf.name
] = _write_once_631mf(
    markdown_path_631mf,
    markdown_631mf.encode("utf-8"),
)


# =============================================================================
# 7) BUILD FINAL STRENGTHENING MANIFEST
# =============================================================================

FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_631MF = {

    "stage":
        "6.31M-F",

    "status":
        "PUBLICATION_STRENGTHENING_REPORTING_EVIDENCE_FROZEN",

    "empirical_authority":
        "STEP_6.31L_REMAINS_AUTHORITATIVE",

    "upstream_final_manifest_sha256_631l":
        str(
            FD001_FD004_FINAL_MANUSCRIPT_ASSEMBLY_MANIFEST_SHA256_631L
        ),

    "upstream_fd002_serialization_audit_sha256":
        str(
            FD002_RISK_SERIALIZATION_AUDIT_SHA256_631MER1
        ),

    "upstream_publication_analysis_contract_sha256":
        str(
            FD001_FD002_PUBLICATION_ANALYSIS_CONTRACT_SHA256_631MER2
        ),

    "upstream_tie_sensitivity_contract_sha256":
        str(
            FD001_FD002_TIE_SENSITIVITY_CONTRACT_SHA256_631MER3
        ),

    "scientific_recomputation":
        False,

    "new_analysis_at_persistence_stage":
        False,

    "raw_test_access":
        False,

    "raw_rul_access":
        False,

    "model_inference":
        False,

    "mc_dropout":
        False,

    "prediction_regeneration":
        False,

    "label_reconstruction":
        False,

    "threshold_tuning":
        False,

    "policy_selection":
        False,

    "primary_result_changed":
        False,

    "fd003_reconstructed":
        False,

    "fd004_reopened":
        False,

    "interpretation":
        PUBLICATION_STRENGTHENING_INTERPRETATION_631MF,

    "artifacts":
        artifacts_631mf,
}


FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_SHA256_631MF = (
    hashlib.sha256(
        json.dumps(
            FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_631MF,
            sort_keys=True,
            separators=(",", ":"),
            default=_jsonable_631mf,
        ).encode("utf-8")
    ).hexdigest()
)


manifest_path_631mf = (
    output_root_631mf
    / "FINAL_PUBLICATION_STRENGTHENING_MANIFEST_631MF.json"
)

manifest_write_631mf = _write_once_631mf(
    manifest_path_631mf,
    _json_bytes_631mf(
        FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_631MF
    ),
)

FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_FILE_SHA256_631MF = (
    manifest_write_631mf[
        "sha256"
    ]
)


# =============================================================================
# 8) CLOSURE CHECKPOINT
# =============================================================================

FD001_FD004_PUBLICATION_STRENGTHENING_CHECKPOINT_631MF = {

    "stage":
        "6.31M-F",

    "status":
        "PUBLICATION_STRENGTHENING_LAYER_CLOSED",

    "empirical_phase_reopened":
        False,

    "original_631l_closure_preserved":
        True,

    "reporting_strengthening_closed":
        True,

    "ready_for_manuscript_integration":
        True,

    "manifest_path":
        str(
            manifest_path_631mf
        ),

    "manifest_semantic_sha256":
        str(
            FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_SHA256_631MF
        ),

    "manifest_file_sha256":
        str(
            FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_FILE_SHA256_631MF
        ),

    "next_phase":
        "manuscript_and_related_work_integration",
}


checkpoint_path_631mf = (
    checkpoint_root_631mf
    / "fd001_fd004_publication_strengthening_631mf_v1.json"
)

checkpoint_write_631mf = _write_once_631mf(
    checkpoint_path_631mf,
    _json_bytes_631mf(
        FD001_FD004_PUBLICATION_STRENGTHENING_CHECKPOINT_631MF
    ),
)

FD001_FD004_PUBLICATION_STRENGTHENING_CHECKPOINT_FILE_SHA256_631MF = (
    checkpoint_write_631mf[
        "sha256"
    ]
)


# =============================================================================
# 9) FINAL FLAGS
# =============================================================================

FD001_FD004_PUBLICATION_STRENGTHENING_LAYER_CLOSED_631MF = True
FD001_FD004_READY_FOR_MANUSCRIPT_INTEGRATION_631MF = True
all_checks_passed_631mf = True


# =============================================================================
# 10) REPORT
# =============================================================================

print("=" * 176)

print(
    "STEP 6.31M-F — FREEZE PUBLICATION-STRENGTHENING "
    "REPORTING EVIDENCE"
)

print("=" * 176)

print("Scientific recomputation          :", False)
print("New analysis in this step         :", False)
print("Raw TEST access                   :", False)
print("Raw RUL access                    :", False)
print("Model inference / MC dropout      :", False)
print("Prediction regeneration           :", False)
print("Label reconstruction              :", False)
print("Threshold/policy selection        :", False)
print("Primary held-out results changed  :", False)
print("FD003 reconstructed               :", False)
print("FD004 reopened                    :", False)
print("Original 6.31L closure preserved  :", True)

print(
    "\nOutput directory:",
    output_root_631mf,
)

print(
    "\nPersisted artifacts:"
)

for name_631mf, info_631mf in (
    artifacts_631mf.items()
):

    print(
        " ",
        name_631mf,
        "|",
        info_631mf["write_status"],
        "| SHA-256:",
        info_631mf["sha256"],
    )


print(
    "\nFinal strengthening manifest:",
    manifest_path_631mf,
)

print(
    "Manifest semantic SHA-256:",
    FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_SHA256_631MF,
)

print(
    "Manifest file SHA-256    :",
    FD001_FD004_PUBLICATION_STRENGTHENING_MANIFEST_FILE_SHA256_631MF,
)

print(
    "\nClosure checkpoint:",
    checkpoint_path_631mf,
)

print(
    "Checkpoint SHA-256:",
    FD001_FD004_PUBLICATION_STRENGTHENING_CHECKPOINT_FILE_SHA256_631MF,
)

print(
    "\nPublication strengthening layer closed:",
    FD001_FD004_PUBLICATION_STRENGTHENING_LAYER_CLOSED_631MF,
)

print(
    "Ready for manuscript integration       :",
    FD001_FD004_READY_FOR_MANUSCRIPT_INTEGRATION_631MF,
)

print(
    "All checks passed                      :",
    all_checks_passed_631mf,
)

print(
    "\nPASS — publication-strengthening reporting evidence "
    "has been persisted and frozen."
)

print(
    "The empirical pipeline remains closed. "
    "Next phase: manuscript + related-work integration."
)